# Fused residual add and RMSNorm

In a pre-norm LLM block, every sublayer starts by adding the previous sublayer's output to the residual stream and normalising the sum with RMSNorm:

$$h = x + r, \qquad y_{ij} = \frac{h_{ij}}{\sqrt{\frac1N \sum_k h_{ik}^2 + \varepsilon}}\; w_j$$

Inference engines fuse both steps into one kernel that returns the updated residual stream $h$ (needed by the next residual add) and the normalised $y$ (fed to attention or the MLP).

Write `add_rms_norm(x, residual, w, eps)`.

## Inputs
- `x`, `residual`: contiguous `float32` CUDA tensors `(M, N)` with $1 \le N \le 8192$.
- `w`: `float32` `(N,)`. `eps`: Python float (e.g. `1e-6`).

## Output
- A tuple `(y, h)` of new `float32` `(M, N)` tensors. The inputs are not modified.

## Requirements
- One program per row, whole row in one block (`BLOCK = triton.next_power_of_2(N)`): read `x` and `residual` once, write `h` and `y` once.
- Note what RMSNorm leaves out compared with LayerNorm: no mean subtraction and no bias.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from a float64 PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def add_rms_norm(x, residual, w, eps):
    """x, residual: (M, N), w: (N,), float32 CUDA. Return (rmsnorm(x + residual) * w, x + residual)."""
    raise NotImplementedError


@triton.jit
def _add_rms_norm_kernel(x_ptr, r_ptr, w_ptr, y_ptr, h_ptr, N, stride_row, eps, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 2.0]], "dtype": "float32"}, {"__nd__": [[2.0, 0.0, 2.0]], "dtype": "float32"}, {"__nd__": [1.0, 1.0, 1.0], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[0.9649012088775635, 0.6432675123214722, 1.2865350246429443]], "dtype": "float32"}, {"__nd__": [[3.0, 2.0, 4.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[3.0, -4.0], [1.0, 1.0]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0], [0.0, 0.0]], "dtype": "float32"}, {"__nd__": [1.0, 0.5], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[0.8485280871391296, -0.5656853914260864], [0.9999995231628418, 0.4999997615814209]], "dtype": "float32"}, {"__nd__": [[3.0, -4.0], [1.0, 1.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.0, -1.0, 0.5, 0.0]], "dtype": "float32"}, {"__nd__": [[-1.0, 1.0, -0.5, 0.0]], "dtype": "float32"}, {"__nd__": [1.0, 1.0, 1.0, 1.0], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}, {"__nd__": [[0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.10468166321516037, -0.8065453171730042, 0.8919731378555298, -0.9285507798194885, -1.1356676816940308, 0.5134580135345459, 1.319050669670105, 1.2758177518844604, 1.3556002378463745, 0.40296003222465515, -0.34895575046539307, 0.5779352188110352, 0.5794493556022644, 0.0661078616976738, -0.6986229419708252, -0.17757895588874817, 0.26550811529159546, -1.155428171157837, -0.36372479796409607, -1.2169758081436157, 1.7756662368774414, 0.7107280492782593, -0.14065319299697876, 0.9527892470359802, 0.17559249699115753, -0.5476267337799072, 0.028777964413166046, -0.1313570886850357, 3.641681671142578, -0.9858279228210449, -2.7848196029663086, 0.7665310502052307, -1.3263137340545654, -1.3989546298980713, 1.3564752340316772, -0.8927116394042969, -1.3230607509613037, 0.01835167594254017, -1.2653939723968506, 0.7417453527450562, -0.7936019897460938, 0.09403914213180542, 1.506269931793213, 2.020174980163574, -2.2683587074279785, -0.5005839467048645, -0.4062609076499939, 0.8529298305511475, -0.43904149532318115, -0.9312676787376404, 1.5131419897079468, 1.3407573699951172, 0.13185586035251617, -2.1364974975585938, -0.14161764085292816, -1.4062128067016602, 2.0727298259735107, 0.5134504437446594, 0.3769960105419159, 0.8980483412742615, 0.7343261241912842, -1.2340610027313232, 0.634585976600647, 0.3554089367389679, -2.610100746154785, 1.208814263343811, 0.4471503794193268, -0.0946047306060791, 0.5522430539131165, -0.8403838872909546, 1.0841730833053589, -0.3856477737426758, -1.293161392211914, 2.119006872177124, 0.8554474115371704, 0.08914987742900848, 0.30090832710266113, 0.5626627802848816, 1.538124680519104, 0.7753117084503174, 1.4379879236221313, -0.6563751697540283, -0.7444276213645935, 0.25438451766967773, 0.4786055386066437, -0.028811965137720108, -0.8431254625320435, -0.2960953414440155, 0.9279054403305054, 0.8590631484985352, -1.1776049137115479, 0.8619683980941772, 0.49836277961730957, 0.40042442083358765, 0.9738825559616089, 0.3124775290489197, 1.8074710369110107, -0.491681307554245, -0.9381097555160522, 0.3162057101726532, 0.498546838760376, -1.062974452972412, 0.5288833379745483, 0.3239125907421112, -1.1330124139785767, -0.29039397835731506, 1.1363911628723145, 1.0072283744812012, -1.0945528745651245, 0.8100159168243408, 0.08211134374141693, 2.198730707168579, -0.18404521048069, 0.66422039270401, -1.5809545516967773, 1.1641223430633545, -0.015859873965382576, -0.548403799533844, -0.49963244795799255, -0.14341841638088226, -0.573279857635498, 0.6214244961738586, -0.4837692379951477, -0.2837143838405609, 3.0221776962280273, -0.045127883553504944, 0.511593759059906, 0.31080323457717896, -0.05607115849852562, 0.1421217918395996, -1.3477555513381958, 0.7503446936607361, 0.2481265515089035, 0.1531074196100235, -1.1415971517562866, 0.9608229994773865, -1.4083664417266846, -0.09069685637950897, -0.7007060050964355, -1.4378266334533691, 1.5623375177383423, 1.4338330030441284, 1.2869844436645508, -0.006138356868177652, -0.8896258473396301, 0.1835116595029831, -0.14141488075256348, -1.8243428468704224, 1.2042388916015625, 0.8366813659667969, -0.6325790882110596, -1.0399388074874878, -0.07576574385166168, 0.21080642938613892, 2.196359634399414, 0.4142724275588989, -1.0330719947814941, 0.6605967879295349, -1.2315905094146729, -1.052594780921936, 0.5111566781997681, -0.22079677879810333, -1.4502513408660889, -0.7081471085548401, -0.48418334126472473, -1.0513579845428467, -0.4927869439125061, 0.1285572350025177, -0.2700670063495636, -0.880038321018219, 0.7451338171958923, 1.5828863382339478, 0.8827695846557617, -1.0951634645462036, -0.8214902281761169, 0.09469897300004959, 0.1831129938364029, -0.9538848400115967, 0.48369696736335754, 0.021739980205893517, -1.2312356233596802, -1.772970199584961, 0.005742082372307777, -0.6701810359954834, 1.9603848457336426, -0.949763834476471, -0.5993463397026062, 2.8347551822662354, -1.178462266921997, 1.0377352237701416, 0.19423553347587585, -0.29859885573387146, -1.081479787826538, 0.14411364495754242, 1.579131841659546, 0.8922150135040283, 0.007472758647054434, 0.9403536319732666, -0.35181954503059387, 0.20487911999225616, 1.2150535583496094, 0.09418055415153503, 0.8932214975357056, 0.14734315872192383, -1.050977110862732, 1.3456121683120728, -0.7042168378829956, 1.3721213340759277, 1.1029667854309082, 1.4536974430084229, 0.8066694140434265, -0.15114876627922058, -0.902579665184021, -0.5112901926040649, -0.5131332874298096, 1.0701137781143188, -1.655621886253357, 0.998753011226654, -0.2515871524810791, 0.46067750453948975, 0.9378744959831238, 0.6399665474891663, 1.0816839933395386, -0.6805874705314636, -0.42184802889823914, -1.3269211053848267, -0.19838397204875946, -0.11144847422838211, 0.41362035274505615, 0.6052985191345215, 1.241477131843567, 1.3237777948379517, -0.5707613825798035, -0.18363605439662933, -0.38186416029930115, -0.28560498356819153, 1.1403928995132446, 1.5626875162124634, 0.7494915723800659, 0.5809842348098755, -0.8456148505210876, -0.5039567351341248, -0.08726052194833755, -2.330630302429199, 0.4164133667945862, -0.15427318215370178, 1.8573768138885498, -0.11370290070772171, -1.6718770265579224, 0.14722216129302979, -0.1335310935974121, 0.28642329573631287, -1.1096338033676147, 0.05606847256422043, -1.7908542156219482, 0.39611345529556274, -0.14989669620990753, -2.2803027629852295, 0.7358688116073608, 1.1031534671783447, 1.1837184429168701, 0.7322801351547241, -0.5860660672187805, -0.7147589921951294, 0.5303341150283813, 1.1479816436767578, 2.82061505317688, 0.3552108407020569, -1.7482285499572754, -0.3092803955078125, 0.8671424984931946, 0.6636236906051636, 1.0781539678573608, -0.8887432813644409, -0.4095929265022278, 0.7181432843208313, 0.4380524754524231, 0.417124480009079, -0.5995995998382568, -0.47020837664604187, 0.5284940004348755, 0.9218194484710693, -1.2731924057006836, 0.2241811901330948, 0.602205753326416, 0.3198188245296478, 0.35104963183403015, -0.07710755616426468, -0.7346887588500977, 1.1738203763961792, -1.276419997215271, 0.29217827320098877, -1.1644463539123535, -0.6373396515846252, -0.14950720965862274, -1.0626459121704102, 0.070877306163311, 0.8445547223091125, 1.1845312118530273, -0.4643956124782562, 0.2943641245365143, -0.5066274404525757, -1.1367496252059937, -0.5947785377502441, 2.014265298843384, -1.052378535270691, -0.6117310523986816, -0.36774393916130066, 0.9766630530357361, 0.10409028083086014, -0.5775114893913269, -0.6425269842147827, 0.2063571959733963, 0.2009143978357315, -1.8163305521011353, 0.06826216727495193, 0.1507311463356018, 0.3411676287651062, -0.3959938883781433, 0.47215306758880615, -1.1700217723846436, 0.20828677713871002, 2.767233371734619, 0.42802467942237854, -0.9328165650367737, 0.6026407480239868, 0.9940878748893738, 0.7562634348869324, -0.3287214934825897, 0.12024721503257751, 0.2932545244693756, 3.2393689155578613, -1.8991639614105225, -0.3693033456802368, -0.2717113792896271, -0.33433184027671814, 0.08749683946371078, 1.1389076709747314, -1.253661870956421, -0.18577660620212555, -0.986648440361023, 0.6215872764587402, 1.8750722408294678, -1.0035649538040161, -0.1501755714416504, -0.38370102643966675, -0.9633074998855591, -0.4386160373687744, -0.060737013816833496, -1.2880172729492188, 0.7598475217819214, -0.9382107257843018, -2.1720163822174072, 0.11253824084997177, -0.25931864976882935, 0.6896458268165588, -0.03928720951080322, 0.30010396242141724, 0.533517062664032, -1.2597575187683105, 1.5364747047424316, 0.9192735552787781, 1.2106959819793701, -0.7380468249320984, 0.08760391175746918, 0.19852450489997864, -1.3605144023895264, 0.590932309627533, 0.4972577393054962, 0.5038295388221741, 0.2191881388425827, 1.0774073600769043, 0.5104904770851135, 0.36230775713920593, 0.8126958608627319, -0.19997847080230713, -0.38948842883110046, 0.8263952136039734, -0.10470566898584366, 0.3211789131164551, -1.4646482467651367, 0.25281956791877747, -1.7518723011016846, -0.6221041679382324, -0.19581711292266846, 2.4262242317199707, -0.2566211223602295, 0.7582154273986816, -0.5624875426292419, -0.307936429977417, -1.6442822217941284, 1.0800663232803345, 0.5630863308906555, 0.7937012910842896, 0.38054803013801575, -0.07341064512729645, 0.6159693002700806, -0.06521376222372055, 1.4636621475219727, 0.567226767539978, -0.352751761674881, -1.1573970317840576, -0.18216250836849213, -1.1056807041168213, 0.4299522638320923, -0.07396101951599121, -1.661865234375, 1.1680203676223755, -0.6310774087905884, -1.8840402364730835, 1.3508014678955078, -0.3295409083366394, -0.5509608387947083, -0.5539348125457764, 0.07061620056629181, -0.09335260838270187, -1.2369695901870728, 0.009768083691596985, 0.013084993697702885, 0.3951994776725769, -1.6142821311950684, -0.3886035084724426, -1.4030207395553589, -0.6307106018066406, -1.2179241180419922, 0.37708139419555664, -0.14228500425815582, -0.15750755369663239, 0.17005737125873566, -0.9108189940452576, -1.3301247358322144, -1.234539270401001, 1.0696539878845215, -0.32135218381881714, 2.4105823040008545, 0.7905035018920898, 0.08174345642328262, 0.8690222501754761, -0.6824533343315125, 0.19018012285232544, -0.035041049122810364, 0.9404796957969666, -1.5313310623168945, -0.9750115275382996, 0.511040449142456, 0.8501442074775696, 1.540849208831787, -0.050916798412799835, 0.13137896358966827, -1.3467803001403809, 0.5204359889030457, 0.6325441002845764, -0.25094446539878845, 0.34878888726234436, 0.21161293983459473, 0.48594745993614197, 0.10489361733198166, 1.4968355894088745, 2.092280864715576, -0.15676814317703247, -0.10533984750509262, -1.2755727767944336, -0.029639912769198418, 1.2838635444641113, -0.9713876247406006, -0.781018078327179, 1.674926519393921, -1.7482104301452637, 1.105424165725708, 0.6243992447853088, 0.030694516375660896, -0.6137710213661194, 0.30934861302375793, 1.3846690654754639, -1.2575045824050903, 1.635642409324646, 0.935398280620575, -0.5211971998214722, 0.008182935416698456, 0.9037092924118042, -1.2529857158660889, -1.5505398511886597, -1.1246784925460815, -1.1535143852233887, 0.11952333897352219, 0.9256125092506409, -0.8046424984931946, -1.4331399202346802, 0.051618561148643494, -0.7243105173110962, -0.3297517001628876, 0.15611572563648224, 0.5881834626197815, 0.23817560076713562, 0.3914220333099365, -0.07532403618097305, 1.2948793172836304, -0.8720994591712952, 2.9856691360473633, 0.07972465455532074, 0.6434237957000732, 0.35276010632514954, -1.1015760898590088, 0.15129421651363373, 0.9535216093063354, 1.0080902576446533, 0.6045935750007629, -0.20919518172740936, -0.6299582719802856, -1.0289759635925293, 1.3683127164840698, 1.5899829864501953, -0.4153262674808502, 0.4414571225643158, -0.165479376912117, -0.014661715365946293, -0.2579784691333771, 1.2471941709518433, 0.2254936248064041, -0.3808791935443878, 1.2829291820526123, -1.388672947883606, 0.5295708179473877, 0.2717956304550171, -0.9577822685241699, -1.7737113237380981, -1.6600924730300903, -0.47839444875717163, -0.755990743637085, -2.0770208835601807, -0.3052842319011688, -0.614726185798645, -0.31076890230178833, -0.3798964321613312, -0.42206257581710815, -0.8086985349655151, 0.23742814362049103, -0.3201155960559845, 1.3099899291992188, 0.0881139487028122, -1.1557451486587524, 2.1612210273742676, 0.3317468464374542, 0.9178056716918945, -0.05774283781647682, -1.2706671953201294, 0.6617989540100098, 0.4929293692111969, -1.7829803228378296, -1.0635671615600586, 0.7350525856018066, -0.5688027143478394, -0.39177191257476807, 0.6036184430122375, 2.1529290676116943, 0.16488562524318695, 1.7414696216583252, -1.2160755395889282, 0.8843459486961365, -0.008245948702096939, -2.1010775566101074, -0.08117183297872543, -0.645389199256897, -0.6562961339950562, 2.7867233753204346, -2.1319637298583984, 0.13287299871444702, -2.0905141830444336, 0.37674590945243835, -0.17339301109313965, 0.9877891540527344, 0.38396990299224854, 0.10706045478582382, -0.6918378472328186, -0.35605019330978394, 1.2406010627746582, 0.89149409532547, -1.1277912855148315, -0.23854972422122955, 1.0470967292785645, 0.8922582864761353, 2.063248872756958, -0.8073171377182007, 0.6653221845626831, -0.2514828145503998, 0.5270697474479675, -0.0007556338096037507, 0.8059741854667664, 1.2608410120010376, -0.7005738019943237, -2.5143799781799316, 0.5987846851348877, 0.03030473180115223, -0.48349907994270325, 0.8494659662246704, 2.3144137859344482, 0.4772166609764099, -0.7221522331237793, -0.320161372423172, -0.48822781443595886, -0.8037633895874023, -0.7024321556091309, -0.5984635353088379, -0.5508480072021484, 0.20769049227237701, -0.24374067783355713, 0.12761612236499786, -1.1320624351501465, -0.8625273704528809, 0.7456511855125427, -1.6820935010910034, -0.023605255410075188, -0.3253891170024872, -0.8326120972633362, 0.04418593645095825, -0.25519657135009766, -1.3629497289657593, 1.0088375806808472, 0.7087444067001343, -2.056272029876709, 0.10337892919778824, 1.1063052415847778, 2.3170523643493652, 1.2930908203125, -1.5029407739639282, -0.5223432183265686, -0.6313387155532837, -0.9182599186897278, -1.6317548751831055, 0.10928165912628174, 0.16366131603717804, -0.487255722284317, 0.4165632426738739, -0.3985348343849182, 1.1430633068084717, 0.6614748239517212, 0.6557782292366028, -0.4421502649784088, -2.037411689758301, -0.37099966406822205, 0.2739964723587036, -2.992950916290283, -0.42882636189460754, 1.9277065992355347, -0.9080404043197632, -0.44982272386550903, 0.9932695627212524, 1.6541045904159546, -0.530976414680481, 1.7587004899978638, -0.0757729709148407, -0.02658357284963131, -0.7783015370368958, -0.013773570768535137, 1.2538964748382568, -1.5252680778503418, -0.3134147822856903, -1.5181059837341309, -0.4879855513572693, 0.04058803990483284, 0.18162228167057037, -0.5428044199943542, 0.40202847123146057, -0.29835018515586853, 0.9574355483055115, -1.6955903768539429, -1.704715609550476, 1.8357759714126587, -1.3781814575195312, -2.7330470085144043, -1.6292424201965332, 0.0033624055795371532, 0.468802809715271, 2.0839829444885254, 2.136002779006958, -0.2260080724954605, 1.2487226724624634, 0.22215671837329865, 0.20735014975070953, 1.0473995208740234, 1.3388820886611938, -0.18966270983219147, 1.3063095808029175, 0.5927284359931946, 0.22325055301189423, -0.4908052980899811, 0.5704984664916992, 1.1109225749969482, 0.5712917447090149, -0.048134855926036835, 1.0400283336639404, 0.46682208776474, -0.8751986026763916, 1.275557041168213, -2.723231077194214, -1.1791741847991943, -1.605234980583191, 0.5422102808952332, 0.18843288719654083, 0.15991508960723877, 0.06183980777859688, -1.5931495428085327, -0.6932857036590576, 1.3097752332687378, 0.5446407794952393, -0.5921136140823364, -1.7616158723831177, 1.053966760635376, 0.3072921335697174, 0.4785695970058441, 0.25150731205940247, 0.5633454322814941, -3.1280057430267334, 0.4177548587322235, 0.7365785241127014, -0.9565303325653076, 1.8995046615600586, -0.5496648550033569, -0.7642958760261536, 0.5289381742477417, -0.5174562335014343, -0.9957733154296875, -0.4552604854106903, -0.6968389749526978, -0.524337649345398, -0.07812348008155823, 0.25273552536964417, -1.620972752571106, 0.5061360597610474, 0.37423449754714966, 1.343759536743164, 0.9135084748268127, -0.6850994825363159, -0.4764286279678345, -0.2838996350765228, -0.1346769779920578, -0.5129191279411316, 0.51651930809021, -0.04702385514974594, 0.24304139614105225, -1.336513876914978, -2.088684320449829, 0.1433308720588684, 0.05988086760044098, -1.66446852684021, -0.6378152370452881, -0.6708289980888367, 0.04170143976807594, -0.16931498050689697, 0.2830716669559479, -1.0673376321792603, -0.3382124602794647, -0.4455571174621582, 0.27410003542900085, 0.3906048536300659, -0.2802351415157318, -1.7583972215652466, -0.20932427048683167, -1.6525171995162964, -0.8883363604545593, -0.38854801654815674, -0.3856944441795349, 2.2428956031799316, 1.444159746170044, 1.3956067562103271, 0.7098613381385803, -1.0122216939926147, 0.7245346307754517, -0.49633216857910156, 0.07886707037687302, -1.4859774112701416, -0.42789337038993835, 1.1303259134292603, -2.3149425983428955, 0.6776478290557861, -0.49064385890960693, -0.7105449438095093, 1.3481594324111938, -0.6676077246665955, 0.8159152865409851, 0.3869756758213043, -0.25420981645584106, 0.3990110754966736, -0.5847289562225342, 1.404101848602295, 0.9579253792762756, -1.4980522394180298, -0.6298633217811584, -0.03554855287075043, 0.07435876131057739, 0.9187221527099609, -0.983961820602417, -1.3671116828918457, 0.06673331558704376, -0.3153831958770752, 0.6341211199760437, 1.670454502105713, -0.8329193592071533, -1.049493670463562, -1.1873140335083008, 1.3047987222671509, 0.8324283957481384, -0.1992729753255844, 0.37046656012535095, -1.2783170938491821, 0.6288012266159058, -0.26093393564224243, -0.30345895886421204, 0.03385864570736885, 0.1680319756269455, 0.08642609417438507, -2.964627742767334, 1.1087850332260132, -1.5251621007919312, -0.8398411273956299, 0.23912771046161652, -0.785703182220459, -1.0574543476104736, 0.10469824075698853, 0.7376207113265991, -1.1420300006866455, -0.2935839295387268, -1.6992006301879883, -0.05259154736995697, 1.97209632396698, 0.9478409290313721, -0.6189083456993103, -0.018977073952555656, 0.090033158659935, 0.03775369003415108, 0.38415250182151794, 1.0024759769439697, 0.43446943163871765, -0.040877778083086014, 0.4497784376144409, -0.46757277846336365, -0.7677901387214661, -1.5298657417297363, 0.5973014831542969, 0.32321590185165405, 0.5609661340713501, -0.4103582203388214, 1.0977182388305664, 0.9290080070495605, 0.2641947567462921, 1.082461953163147, -0.023888874799013138, 0.7961979508399963, -0.7250837683677673, 0.932643473148346, -1.464530110359192, -0.8801568746566772, 0.04014096036553383, 1.0927095413208008, 0.049299608916044235, -1.1128580570220947, 0.4755770266056061, -2.427096366882324, 0.012472515925765038, 0.08550367504358292, -0.12054150551557541, 0.36305972933769226, 0.3028390407562256, 0.03588494658470154, -0.6738379001617432, -0.435993492603302, 0.38722753524780273, -1.360336422920227, -1.1447724103927612, -0.04827500507235527, -0.22447921335697174, 0.6594240069389343, 0.8158895373344421, -0.21176323294639587, -0.6853511929512024, 0.8659071922302246, -1.602799654006958, 0.8035513162612915, 1.7187212705612183, -0.4763312041759491, 0.7732482552528381, 0.500115156173706, -0.6317205429077148, -0.6671774387359619, 0.9716262817382812, -0.12070556730031967, -0.8946854472160339, -2.2892532348632812, -0.26289376616477966, 0.007130964659154415, 0.4367397725582123, -0.10163036733865738, -1.454555869102478, -1.333044409751892, -0.7876719832420349, 0.05502142757177353, -0.6648480296134949, 0.28495439887046814, 0.6659325361251831, 0.29802194237709045, 0.7307451367378235, 0.46112605929374695, -0.2974490225315094, -0.7389004826545715, 1.502607822418213, 0.4180169999599457, 1.0462473630905151, 1.1345136165618896, 0.39014744758605957, 0.3530483841896057, 1.0083993673324585, -0.43383505940437317, -1.1875985860824585, 0.5542805194854736, 1.1033490896224976, -0.6157613396644592, -0.27821290493011475, 0.4104699194431305, -0.859018862247467, -1.492993712425232, 0.2810693085193634, -0.18030640482902527, -1.5153864622116089, 1.0312395095825195, -0.9521855711936951, -1.4516063928604126, 0.0591321736574173, 0.06602871417999268, 1.842269778251648, 0.4187620282173157, -1.163244605064392, -0.9722167253494263, -0.8067201375961304, 0.39658859372138977, -0.5356863141059875, 0.6440862417221069, -0.40668532252311707, -1.8463362455368042, -2.3479154109954834, -1.1148953437805176, -0.6044748425483704, -0.39346733689308167, 0.7349743843078613, 0.5994810461997986, -1.1183840036392212, 0.24279440939426422, -1.767518162727356, 0.9393317699432373, -0.1251707822084427, -1.3522599935531616, 0.8126201629638672, 0.16589181125164032, 0.8412489891052246, -0.30219200253486633, 0.15823400020599365, 1.0180106163024902, -0.9497727155685425, 0.39488810300827026, -1.070603609085083, 3.1795475482940674, -0.04148134961724281, 0.33975324034690857, 1.5945730209350586, 0.005988189484924078, 0.40653282403945923, 0.1888861209154129, 0.9430111646652222, 0.25644397735595703, 0.41236668825149536, 0.23610465228557587, -0.5564737915992737, 0.7421198487281799, -1.1596970558166504, -1.8563940525054932, 0.42180314660072327, 1.5437475442886353, 1.324867606163025, 0.32083144783973694, -0.946444034576416, -0.03135203197598457, 0.3175634443759918, 1.3918039798736572, 0.23436520993709564, 1.5030736923217773, 0.5003064870834351, 0.48340409994125366, -0.16209739446640015, -0.02731047011911869, -0.6600071787834167, 0.016660654917359352, -0.11380375921726227, 1.3104225397109985, -1.1718063354492188, 0.08312415331602097, -0.050207749009132385, -1.0076088905334473, 1.421728491783142, 0.951816201210022, 0.42787325382232666, -0.6334792375564575, -0.6270087361335754, -0.8134365081787109, -2.0019021034240723, 2.126420497894287, 0.8243647217750549, 0.8862961530685425, -0.7214399576187134, -0.8047484159469604, 0.6803531646728516, -0.6526366472244263, 0.8597277402877808, -1.0227115154266357, -0.4308694005012512, -0.33726766705513, -0.22670899331569672, -1.3083815574645996, 0.4641980826854706, -0.02741391956806183, -2.9739866256713867, -1.9940038919448853, 0.06396452337503433, -0.9023783802986145, -0.7528685331344604, -2.0529704093933105, 0.3754860758781433, 0.35248562693595886, -0.14417722821235657, -0.5336189270019531, -0.3718806803226471, 0.9519839882850647, 0.5178886651992798, 0.5824435949325562, -0.12574021518230438, 0.020585067570209503, -0.2775871455669403, 0.7233420014381409, 0.7289493680000305, -1.9616258144378662, 0.6800057888031006, 1.9298303127288818, 0.48619529604911804, 0.025621477514505386, 0.2315710037946701, -0.1221044659614563, 0.9174463152885437, 0.9443029165267944, -0.5379931330680847, -0.5001683235168457, 1.4005813598632812, -1.313825249671936, -0.13662642240524292, 0.8187440037727356, -0.3619335889816284, 0.17222562432289124, 1.5919777154922485, 0.3833082914352417, 1.1751891374588013, -0.4565652310848236, 0.20578806102275848, -2.322219133377075, 0.6268729567527771, -0.5518076419830322, -0.1923539936542511, 1.0071024894714355, -2.109341859817505, 0.1854449361562729, 0.9098367094993591, 0.95663982629776, -2.0233266353607178, 0.5448036789894104, 1.3278546333312988, -0.9851896166801453, 0.48676422238349915, 0.48642972111701965, -0.23134200274944305, -0.19637636840343475, 1.2644795179367065, 0.40314429998397827, -0.180122509598732, -0.9893970489501953, 1.1749540567398071, 2.2566864490509033, 1.4686208963394165, -2.347317934036255, -1.2463041543960571, 0.3099433183670044, 0.14120705425739288, -0.16195844113826752, -2.247169017791748, -0.9106709361076355, 0.49273496866226196, -0.906436562538147, -1.4499353170394897, -0.3061593472957611, 0.2629493176937103, 0.4885641932487488, -0.9669395089149475, -1.1329981088638306, -0.16073377430438995, 0.3042134940624237, 1.0176364183425903, -0.9840990304946899, -0.5231059193611145, -1.8025795221328735, -1.6693426370620728, 1.57660710811615, 0.145736962556839, 0.804456889629364, 0.15919823944568634, -0.7718403935432434, 1.0892024040222168, 1.1700375080108643, 0.4181826412677765, 1.4035879373550415, 0.23203034698963165, -0.05100543051958084, -0.034589026123285294, 1.642909049987793, -0.5101089477539062, 1.3900439739227295, 1.2103828191757202, -0.3260663151741028, 2.6671183109283447, -0.10876362770795822, -0.8027604818344116, 1.0832964181900024, -0.24986036121845245, 0.7562732100486755, -0.13955754041671753, -1.553619146347046, 0.759016215801239, -0.5405297875404358, 0.651075541973114, -0.20479990541934967, -0.6427812576293945, 0.03975893557071686, 0.9559571146965027, 0.5527997612953186, -0.24516801536083221, -2.525355100631714, 0.7946208119392395, 0.08896427601575851, 0.8554133176803589, -0.33507707715034485, -0.07507558912038803, -0.3121242821216583, 0.26332008838653564, -0.08722271025180817, -0.4093080461025238, 1.8730136156082153, -0.7588939070701599, 0.7947620749473572, 0.4831961691379547, -1.281440019607544, 1.9183346033096313, -0.2124776691198349, -0.5433036088943481, 0.6034847497940063, 0.09439679980278015, -0.2875528335571289, -0.26370111107826233, -0.34035196900367737, -2.0223143100738525, 1.5351260900497437, 1.0954921245574951, -0.23191797733306885, -0.9212136268615723, 0.34487324953079224, 1.084376335144043, 0.9405297040939331, 0.486558198928833, -0.10963659733533859, 0.20566895604133606, 0.2526969015598297, 0.05455775558948517, 0.5555232763290405, 0.7086456418037415, 1.2119545936584473, 1.0390383005142212, -0.1567617505788803, 0.10046829283237457, -0.4904486835002899, 0.4502304792404175, -0.8104223012924194, 1.679840326309204, -1.1647846698760986, 0.39639660716056824, 0.44976431131362915, 0.5868809223175049, 2.9891903400421143, -0.3453468680381775, -0.39630889892578125, 0.507346510887146, -0.8636246919631958, 0.3899727165699005, 2.788048028945923, 0.08329339325428009, 0.2568548321723938, 0.43716174364089966, -0.05340449512004852, -0.7735477685928345, -1.3057154417037964, 1.8039861917495728, -0.21965591609477997, 0.3362087607383728, -0.9244131445884705, -0.626593828201294, 0.5715702772140503, -0.4028361737728119, 0.9135953783988953, 0.8058263063430786, -0.8755656480789185, 1.4999866485595703, -0.11816059798002243, 0.9673881530761719, -1.4313210248947144, -0.12003026157617569, 0.9585743546485901, -0.026693129912018776, -0.04468448460102081, -0.06471893936395645, 2.1490702629089355, -0.3245409429073334, 0.6320807933807373, -0.5381097793579102, -0.383578360080719, 0.5740225315093994, 1.2868865728378296, -1.5695816278457642, 0.5107666850090027, 0.19681452214717865, 0.026298601180315018, -0.021379247307777405, -0.9622975587844849, 0.9332522749900818, -0.6319559216499329, -0.3549959063529968, -0.48573288321495056, -1.2120885848999023, -0.3851938545703888, -1.376354455947876, -0.9344764351844788, 0.40188437700271606, -0.32389092445373535, -0.20764216780662537, -0.4909537136554718, 0.5215466022491455, 0.7770124673843384, 0.02942526713013649, 0.6949523687362671, -1.836419939994812, 0.5129597783088684, 0.17022588849067688, -0.5007901191711426, 1.0383484363555908, 0.8021070957183838, -0.36663246154785156, -0.2617701292037964, 0.36271145939826965, -0.37193745374679565, -0.5687625408172607, -1.6746898889541626, 0.8730651140213013, -0.7516005635261536, -0.5054689645767212, 0.37985438108444214, -0.24499565362930298, 0.9657680988311768, -0.0979146808385849, -1.8673925399780273, 0.9601231217384338, 0.25423818826675415, -0.01546180434525013, 0.2834683358669281, -0.2186422348022461, 0.859622597694397, 0.661320686340332, 0.40580013394355774, 0.49477219581604004, -1.523177146911621, 0.4315924048423767, -2.383768081665039, 0.3844493627548218, 1.3804408311843872, 2.0462160110473633, -0.06923765689134598, -0.2055208683013916, -1.342581868171692, 1.624799132347107, -1.6947627067565918, -0.5952433943748474, -0.32226794958114624, -0.7775636911392212, 0.36694324016571045, 0.3450537919998169, 0.135455921292305, 1.3521013259887695, 0.6694877743721008, -0.49809175729751587, -0.7169726490974426, 2.1718616485595703, -1.3404685258865356, -0.3798496723175049, 0.0854981392621994, -1.4832100868225098, 0.5055801868438721, -0.8671334385871887, 1.971063494682312, -0.3715318441390991, 0.7691224813461304, 0.3326915204524994, -0.08465104550123215, -0.43265092372894287, -0.4707046151161194, 1.0865046977996826, 0.7793607711791992, -0.25108757615089417, 0.23296865820884705, -0.7603147625923157, -0.1709664911031723, -0.6086498498916626, -0.47891849279403687, 2.356762409210205, 0.23733220994472504, -1.5883610248565674, 2.107896566390991, -0.44715678691864014, 0.9650080800056458, -0.6561185121536255, 0.039395641535520554, -0.6036797165870667, 0.061589550226926804, 1.4278604984283447, 1.6722804307937622, 2.0452699661254883, -0.3058551549911499, 0.47801679372787476, -0.32433322072029114, -0.15547962486743927, 1.8352606296539307, 0.5427495241165161, -2.321866750717163, -0.12672452628612518, 0.5074414014816284, 0.7670865654945374, -0.47581201791763306, -0.9271156191825867, -1.7602956295013428, 0.8293377161026001, 0.15931633114814758, -1.2283183336257935, 0.7925553321838379, 1.50795578956604, -0.6152002811431885, 1.2835344076156616, -1.1445600986480713, 0.45009085536003113, -0.19814561307430267, 1.0277135372161865, 0.13675495982170105, 1.5419833660125732, 0.03329082950949669, -1.2631491422653198, 0.344074010848999, 0.09253962337970734, 1.0167542695999146, -0.9760038256645203, 0.9548048377037048, -1.0269368886947632, -2.0112075805664062, -0.45075663924217224, 0.05553658306598663, 1.8923943042755127, 0.11567860841751099, -0.797221839427948, 1.4080222845077515, 1.518416404724121, 1.249260663986206, -1.4187119007110596, -0.4773699939250946, 0.9541443586349487, -1.6672111749649048, 0.055854447185993195, 0.04199981689453125, 0.013737411238253117, 0.05825261399149895, 0.3736379146575928, -0.42374491691589355, 1.378548502922058, 0.14193366467952728, 0.7754107117652893, 1.3100290298461914, 1.392555594444275, 0.7724257111549377, 1.7456796169281006, -0.17029798030853271, 0.2378583401441574, -0.07885283976793289, -0.3677898943424225, 0.440060555934906, -0.18821890652179718, 0.15602940320968628, -0.2689659297466278, -1.178993582725525, 0.831200897693634, 1.0348224639892578, 0.3682132661342621, 0.18349507451057434, 0.40185245871543884, 0.1452648788690567, 1.0599623918533325, -0.5450511574745178, -0.11202820390462875, -0.7347216606140137, -1.2554198503494263, -0.6720921397209167, 0.3375609815120697, -0.45568767189979553, -1.4453189373016357, -0.1094651147723198, 0.7909508943557739, 1.6267569065093994, 0.9893428683280945, -0.8020105361938477, 1.0344520807266235, -2.1381866931915283, 0.2458416372537613, 0.7082198262214661, 1.1504642963409424, 0.22312186658382416, -0.797712504863739, 1.1496119499206543, 1.2565094232559204, 0.27656763792037964, -1.5847113132476807, -0.6740804314613342, -0.5137516856193542, -1.3084962368011475, -1.181649088859558, -1.038490891456604, -0.11008056998252869, 0.22820107638835907, 0.9073446393013, 0.27398207783699036, 1.5481576919555664, 1.4335856437683105, -0.2821529507637024, 0.48400819301605225, 0.46994835138320923, -0.034830573946237564, 0.2641181945800781, 0.5649716854095459, -0.29572510719299316, 0.03963874652981758, 0.4375014901161194, -0.49384790658950806, 1.44437575340271, -1.3460843563079834, 1.084398865699768, -0.2669949531555176, -1.9248790740966797, -0.3103373944759369, 0.0360223613679409, -1.3134183883666992, -0.2372247725725174, 0.14984889328479767, -0.8602626919746399, 0.6166486144065857, 0.6099576950073242, -0.633034884929657, -0.4745877981185913, 0.767430305480957, 0.07653787732124329, 0.5999372005462646, -0.5921158790588379, -0.6627402901649475, 0.7002205848693848, 0.1972624510526657, 0.5025166273117065, 1.685814380645752, 0.4048061966896057, -0.1303189992904663, -1.6746212244033813, 0.27698540687561035, 0.5200858116149902, 0.22143813967704773, 1.1771094799041748, 1.5865280628204346, 0.21322549879550934, -0.6809890270233154, -0.45849207043647766, 0.020999249070882797, 1.3756108283996582, 0.16729792952537537, 1.320385217666626, -1.1234748363494873, 1.2470191717147827, 0.6521049737930298, -0.23082496225833893, 0.4470270574092865, 1.1180700063705444, 0.6743082404136658, 1.1417038440704346, 0.6131512522697449, 0.8310059905052185, 1.0453847646713257, 0.1318836659193039, 0.1500466763973236, 0.7562556862831116, 0.7700712084770203, -1.0577754974365234, -0.6119639277458191, -1.1457581520080566, -0.22200323641300201, -0.8088290095329285, 1.37744140625, -1.4322701692581177, -0.27533024549484253, 0.3538517355918884, -1.7304182052612305, 0.9343510866165161, -0.22732006013393402, -0.037871427834033966, 0.5701348781585693, 2.2180540561676025, -0.11502029746770859, -0.47705045342445374, 0.6452286243438721, -1.2337216138839722, 0.5074419975280762, 0.5249574184417725, -1.678446888923645, -0.4883384108543396, 1.575673222541809, 0.7507332563400269, -0.03053421340882778, 1.9606841802597046, -2.074122667312622, -0.06664688140153885, 0.6264083385467529, 0.44504743814468384, 0.5134199857711792, -0.7018601894378662, -0.8498713970184326, 1.9003005027770996, 0.40764179825782776, 0.8183737993240356, -0.7729602456092834, -0.8557814955711365, -0.3554303050041199, 0.9730470180511475, 0.5771521329879761, -0.5115997791290283, -0.5322303771972656, 1.5490999221801758, 0.19931501150131226, 0.4498286843299866, 0.9734077453613281, 1.5507402420043945, -0.22585132718086243, 0.7700297832489014, -0.43982335925102234, 2.0987257957458496, 0.43209967017173767, 0.09472857415676117, 0.2846527695655823, -0.6286530494689941, 0.6450279951095581, 0.0830150917172432, 0.38494327664375305, 0.5659579634666443, -0.2599296271800995, 0.19029048085212708, 1.5583231449127197, 1.9515631198883057, -0.25146129727363586, 0.5221169590950012, -0.312272846698761, 0.3665103018283844, 2.0818166732788086, -1.7254995107650757, -0.871235191822052, 0.9971799850463867, 1.6909842491149902, 0.32263240218162537, -1.3353466987609863, -0.7073681354522705, -0.3008096218109131, 1.571364402770996, 1.1946922540664673, 0.6872475743293762, 3.1719555854797363, -1.646079659461975, 0.27750515937805176, -0.11646474897861481, 1.074398398399353, 0.4323534667491913, -0.45388782024383545, 1.2770843505859375, 0.12034802883863449, -0.4221932291984558, -0.19321979582309723, -1.2799134254455566, 1.0438913106918335, -1.6885536909103394, -0.07683508098125458, 0.8865239024162292, 0.9025394320487976, 0.8326643705368042, 0.40137141942977905, 0.32493162155151367, -1.249244213104248, -0.6594018936157227, -1.4879403114318848, 1.2527153491973877, 0.21113267540931702, 1.5097121000289917, 1.3112448453903198, -0.12459177523851395, -0.29125261306762695, 0.15481680631637573, -0.39323464035987854, 0.45520350337028503, 0.9310211539268494, -0.26385298371315, 1.7679697275161743, -0.762860119342804, -1.1044753789901733, -0.48534300923347473, 0.634936511516571, 0.3864682912826538, -1.0909335613250732, -0.6068747639656067, 1.2402552366256714, 0.5230608582496643, -2.166438102722168, 0.27121421694755554, -0.17498956620693207, 0.14076340198516846, 0.9339569807052612, 1.1140490770339966, 0.4587056338787079, 1.213120937347412, -0.7705376148223877, -0.2677188217639923, 1.113088846206665, -1.673121690750122, 0.14334037899971008, -0.5178122520446777, 0.15837620198726654, -0.03127862885594368, -0.7076389193534851, 0.6308673024177551, -1.2259873151779175, 0.4234137237071991, -0.06235792487859726, -1.3647334575653076, 0.6492087244987488, -0.3041384816169739, 1.0569771528244019, -0.14371201395988464, 0.32022103667259216, 2.201437473297119, -0.0011069821193814278, 0.5571980476379395, 0.7061403393745422, 1.2240092754364014, -0.09437218308448792, -0.09104501456022263, -1.2604745626449585, -0.3134593963623047, -0.4796338379383087, -0.5970747470855713, 0.03774450346827507, 0.6265586018562317, -1.0146541595458984, 1.6636074781417847, 1.305154800415039, 0.3128124475479126, 0.9667807817459106, -1.4836061000823975, -0.301615446805954, -0.07992472499608994, 0.1706649214029312, -0.6357991695404053, 0.07454049587249756, -0.008642049506306648, 1.68630850315094, 0.7228384613990784, -0.04646238312125206, -0.18475459516048431, -0.4743667542934418, 0.07757454365491867, -0.8210821151733398, 0.23187072575092316, 1.7113744020462036, -0.9180041551589966, -2.014450788497925, 1.3465343713760376, -1.1249604225158691, -1.6723394393920898, 0.11680914461612701, -1.0844916105270386, -0.7879948019981384, 0.52069491147995, 0.5671910047531128, 0.38080349564552307, -0.21461006999015808, 0.8695266842842102, 0.43844926357269287, -0.26774147152900696, 1.5032988786697388, -0.2776227593421936, -0.28794264793395996, -0.3020627498626709, 0.15825685858726501, -0.2251136153936386, 0.030152857303619385, -2.6855227947235107, 1.2132396697998047, 0.30714190006256104, 0.744663417339325, -0.7386519312858582, 2.282026767730713, 1.8621960878372192, -1.221285104751587, -0.8150803446769714, -1.6550811529159546, 1.033289909362793, 0.7074235677719116, -0.18007950484752655, 2.45365571975708, -2.3328590393066406, -2.0196080207824707, 1.0254658460617065, 1.4862719774246216, 0.7180656790733337, 1.0522032976150513, 1.0526334047317505, -2.0616137981414795, -0.9282088279724121, -0.6845653653144836, -0.807471752166748, -1.1412076950073242, 0.8644338250160217, 0.17701151967048645, -0.3213983178138733, -1.313594937324524, 0.05538565665483475, 0.5521374344825745, 0.7033114433288574, 1.0724271535873413, 0.5459430813789368, -0.3302827477455139, 0.43206191062927246, -1.4545888900756836, -0.3511306941509247, 1.4028784036636353, 0.29296550154685974, 1.5197256803512573, -0.007910938002169132, -0.6993071436882019, 2.9608891010284424, -0.6653543710708618, 0.15363679826259613, 2.367499589920044, -0.7124156355857849, -1.564648151397705, -0.3651380240917206, 0.8683984279632568, -1.0779801607131958, -1.4984405040740967, -0.17910946905612946, -0.6429880261421204, 0.18525172770023346, 0.3602816164493561, -1.5843188762664795, 1.3146674633026123, -0.5921146869659424, -0.6175894141197205, 0.7943690419197083, 0.12965300679206848, 0.4561786949634552, 0.781883716583252, 0.49988359212875366, 0.24383606016635895, -1.2283368110656738, -0.8607149720191956, -0.5142956376075745, 0.4772038757801056, 0.6820409297943115, 0.4670776128768921, 0.5385591387748718, -0.9665507078170776, -0.48943397402763367, -1.3014566898345947, -0.2621902823448181, -1.454444408416748, -0.46239960193634033, -2.1152312755584717, -0.6192602515220642, -0.41031602025032043, 1.1647032499313354, -0.5226447582244873, 0.48319175839424133, 1.4403891563415527, 1.4172652959823608, 0.4862161874771118, -1.6357319355010986, -0.43220242857933044, 0.8642532825469971, 0.03810169920325279, 0.6876357793807983, -1.0522730350494385, -0.370033860206604, 0.41742342710494995, -1.4410334825515747, 0.2943457067012787, 0.20873069763183594, -0.9846866726875305, 0.7974342703819275, 1.374219298362732, 1.052905559539795, 0.10760953277349472, -0.05393601581454277, 0.3310606777667999, 1.2232416868209839, -0.8754714727401733, 1.406256914138794, -0.21156705915927887, -0.23777155578136444, -0.3528509736061096, 2.098296880722046, -0.724128782749176, 0.9670973420143127, 0.18824365735054016, -0.8238368034362793, -0.5542449951171875, 1.2405563592910767, 0.5639222264289856, -0.1198815330862999, -0.19221188127994537, -0.44248706102371216, 1.1620782613754272, 1.2263851165771484, -1.5188884735107422, 0.2739797532558441, 0.4407263994216919, -1.1116431951522827, -0.572824239730835, -0.23746643960475922, 0.40962475538253784, -0.21150514483451843, -0.3001190721988678, -1.1000005006790161, 0.6853576302528381, -0.038948915898799896, -0.13824652135372162, -0.2924332022666931, -1.0820438861846924, -1.28849196434021, 0.6609392166137695, -0.9237057566642761, -0.6787837147712708, -1.3648922443389893, 0.6923794746398926, 0.7586212754249573, 0.8986443877220154, -0.598563015460968, 0.175231471657753, 1.145768165588379, -0.9840343594551086, -0.9497076272964478, 1.1901081800460815, -0.8419787883758545, -2.332427740097046, -1.5351898670196533, -1.3772436380386353, -0.7532730102539062, -0.3289835751056671, 0.30259934067726135, -0.920737624168396, 0.13436675071716309, 1.06133234500885, 0.3807411789894104, 3.56795072555542, 1.3336410522460938, -0.0629827231168747, 0.18392322957515717, -0.5850589275360107, 0.05539698153734207, 0.12413384020328522, -0.16939441859722137, 0.46009400486946106, 1.1972423791885376, 0.09042444080114365, -0.9146829843521118, 0.6871934533119202, 0.572272777557373, 2.2127630710601807, -0.14643393456935883, -0.6186737418174744, 0.5244547724723816, -0.30424413084983826, 0.4304141700267792, 1.1028499603271484, 0.8142206072807312, 1.2329070568084717, 1.0589667558670044, -0.7123379707336426, 0.16807222366333008, -0.33598390221595764, 0.21161434054374695, -0.9054358005523682, 0.0611727274954319, -0.19992004334926605, 0.6704902052879333, -1.1615660190582275, -1.383392572402954, -1.0069384574890137, 0.009429015219211578, -0.5601580739021301, -0.7176916003227234, -0.8949154019355774, 1.691205620765686, -1.1715891361236572, -0.4636026620864868, 1.3886442184448242, -0.24050793051719666, -0.8866854310035706, -0.7773311734199524, -0.9875670075416565, 1.4897619485855103, 1.4184918403625488, 0.3282889723777771, 1.7440794706344604, -0.41498491168022156, 0.3410469889640808, 0.6407409906387329, 0.3219364881515503, -0.012612254358828068, -0.9388310313224792, 0.38674744963645935, -0.17498895525932312, 0.5438105463981628, -2.2721762657165527, 0.36195576190948486, -0.2933202087879181, 0.34699365496635437, -1.0106279850006104, 0.883223295211792, 1.1741821765899658, -1.0216264724731445, -2.3130147457122803, -1.5588414669036865, -0.5646703243255615, -0.03794748708605766, 0.9644319415092468, 1.3001668453216553, 0.17052499949932098, -1.0348589420318604, 0.3556004762649536, -0.834872305393219, -1.646485447883606, 0.9472529888153076, 0.23379051685333252, -0.23919646441936493, 0.9747881889343262, -0.4694947898387909, 0.4373168647289276, -0.5223269462585449, -0.8887767791748047, 0.951626718044281, -0.7254688739776611, -1.3334901332855225, 0.9523716568946838, 0.5402659773826599, 1.0461560487747192, -1.9050681591033936, 0.29636624455451965, -1.2944202423095703, -0.40244773030281067, -0.26997101306915283, -0.8096369504928589, -0.8026348352432251, -1.338985800743103, -0.2305590659379959, -1.9215677976608276, -2.229032039642334, 0.5666467547416687, 0.9450455904006958, 0.0996093675494194, -0.5653381943702698, -1.0297372341156006, -0.6826286911964417, -1.9779504537582397, 0.09357182681560516, 0.9249580502510071, 1.8011976480484009, 0.3225092887878418, -1.136582851409912, 1.6316698789596558, 1.2258753776550293, 0.5363245010375977, -0.9660948514938354, 0.2496425360441208, -1.1571065187454224, 0.15935957431793213, 0.9544628262519836, -0.4429158866405487, 0.6198605895042419, 0.33259448409080505, -0.13926397264003754, -1.1828991174697876, -1.0023802518844604, -0.6371804475784302, -2.086562395095825, 0.01314813457429409, -0.6543765068054199, 0.2513366937637329, 0.35078948736190796, -0.025717519223690033, 0.786629855632782, -0.9748731851577759, -1.4094657897949219, 0.22761139273643494, 0.2770756185054779, 0.6311420798301697, 0.3472185730934143, 0.32268133759498596, -0.8291063904762268, -1.2927136421203613, 0.16287024319171906, -1.998913288116455, -0.03388554975390434, -0.708509087562561, -2.1368215084075928, -1.3251891136169434, -1.709580898284912, 2.0888783931732178, -1.4796557426452637, 0.3249761760234833, 0.40084972977638245, -0.5103784799575806, -0.3923795223236084, 0.37624579668045044, 1.916684865951538, 1.0203217267990112], [-0.6310632228851318, -1.4188337326049805, 0.6383673548698425, 0.6994829773902893, -1.2520596981048584, -1.7508773803710938, 0.23808439075946808, -1.6417045593261719, 0.06369362026453018, -2.8638784885406494, -0.5121569037437439, -1.5759854316711426, 0.9911691546440125, 1.61319899559021, 0.16826224327087402, 1.0728284120559692, 0.06641809642314911, 0.5722687244415283, 1.806087851524353, -0.5502312779426575, -1.8213324546813965, -0.005148874595761299, 2.283489942550659, 1.2661100625991821, 2.0418550968170166, 0.06255951523780823, -0.31626468896865845, -1.307456135749817, -0.5933854579925537, -0.4379447102546692, -0.39421552419662476, 0.8066501617431641, 0.4053013026714325, 1.3750487565994263, -0.502264678478241, -0.5367162227630615, -0.7482833862304688, 1.0612506866455078, -0.1956312358379364, 0.4204866290092468, 0.13539977371692657, -0.4798395335674286, 0.5814936757087708, 1.3475449085235596, -0.4896393120288849, -1.8439393043518066, 0.9699366688728333, 0.8664035797119141, -0.01889997348189354, 0.9574966430664062, 1.2873728275299072, 1.3165193796157837, 1.5543967485427856, -0.31253302097320557, 0.8773953914642334, -0.8592864274978638, 0.06479943543672562, 1.1706972122192383, 0.2095934897661209, -0.38123857975006104, 0.8329847455024719, 0.8523194193840027, 1.0877456665039062, -2.706838607788086, -1.1539233922958374, 0.3441007435321808, 1.1028186082839966, 0.6258363127708435, -1.1368134021759033, -0.28303438425064087, -0.2807702124118805, -1.7007853984832764, -1.2596533298492432, -0.8495974540710449, -1.2257511615753174, 1.6882396936416626, -0.6997112035751343, 0.1457880735397339, -0.8574202656745911, 0.2530701458454132, -1.2436518669128418, -1.2743386030197144, 1.643984079360962, -0.19217921793460846, -1.1682243347167969, -1.395063042640686, -0.7591588497161865, 0.964364230632782, 0.8291627168655396, 0.9310388565063477, -0.2982458174228668, 0.3286254107952118, 0.20639188587665558, -0.6174749732017517, -0.5954110026359558, 0.2206779420375824, 0.9688808917999268, 1.4513859748840332, -0.9950783848762512, -0.2037268877029419, 0.12189039587974548, 1.601920485496521, -1.1854873895645142, -0.3344455063343048, -1.6301119327545166, -0.934933602809906, -1.1034038066864014, -1.6203292608261108, -0.9925321936607361, 0.5689323544502258, 0.13575169444084167, 2.1829516887664795, -0.15210992097854614, 1.03878915309906, -2.2973177433013916, -1.9982426166534424, 1.1311006546020508, 0.9827473759651184, -0.30308133363723755, -0.8849696516990662, 1.1592388153076172, -0.44191035628318787, -0.3660805821418762, -0.9068123698234558, -0.012338340282440186, 0.7442436814308167, 1.2778009176254272, -0.5288992524147034, 0.1965782195329666, -0.5172207355499268, 1.994565725326538, -1.4643940925598145, -1.6899020671844482, 1.0244262218475342, 0.7385838627815247, 0.6801588535308838, -0.9018722176551819, -0.4713539481163025, 0.501465916633606, -0.45429328083992004, -0.6258816719055176, -0.5558122396469116, 1.8632776737213135, -2.372843027114868, 0.21843993663787842, -0.7540668249130249, -0.790404200553894, 0.30264294147491455, 0.86191326379776, -0.4605056941509247, 2.746856212615967, -2.035175323486328, 0.2844705581665039, 0.5304864645004272, -0.5944528579711914, -0.23502089083194733, -0.5209451913833618, -0.230324849486351, 0.020290235057473183, 0.15869081020355225, -2.5676369667053223, -0.033588092774152756, -0.16466861963272095, -2.6831371784210205, 0.7808482050895691, -0.03698981925845146, 0.8600031733512878, -1.082044243812561, 0.9882801175117493, 0.23126034438610077, 2.1827478408813477, -3.2007222175598145, -1.1996794939041138, -2.9606127738952637, 1.5272208452224731, 0.3435983955860138, 2.1198196411132812, 0.8987952470779419, -0.6163374185562134, 1.6662238836288452, -0.39394882321357727, 0.145126074552536, 2.468116521835327, -0.7738761305809021, 1.0213109254837036, -0.24924099445343018, 0.7684609293937683, 0.19167469441890717, -0.1307147592306137, 1.2904739379882812, 1.498754620552063, -2.0168650150299072, -1.8344247341156006, -0.9334522485733032, 0.7591123580932617, -0.13658258318901062, -1.5346102714538574, 0.34093788266181946, -0.5404829978942871, -0.9504693150520325, -1.224109411239624, -0.2843164801597595, -0.380435585975647, 1.1766571998596191, -1.2726694345474243, -0.13349714875221252, 0.22774624824523926, -0.6215961575508118, 0.4295446574687958, 1.2298095226287842, 0.13670776784420013, -0.2004203051328659, -1.2978317737579346, -0.6769388914108276, -0.11681350320577621, 0.9851891398429871, 0.09830936044454575, -0.62091463804245, 0.13633063435554504, 0.5572341680526733, -0.09827329963445663, -0.4546144902706146, 0.2777976989746094, 0.26668527722358704, -2.0613112449645996, -0.9946417808532715, -1.6116691827774048, 1.7759344577789307, 0.008884531445801258, -0.3736242949962616, -0.462209552526474, -0.15494653582572937, 0.12185267359018326, -1.1171449422836304, 0.6538978219032288, -0.8923137187957764, -1.434787392616272, 2.599130630493164, -1.272359013557434, 0.4020821452140808, 0.18039163947105408, 0.1798269897699356, -0.6347090601921082, -0.22542616724967957, 0.9579179883003235, -0.537982702255249, -0.2019181251525879, 0.23037412762641907, -0.07082907855510712, -1.2485196590423584, 1.4367648363113403, 1.1129387617111206, 0.5287482142448425, -0.14610058069229126, -0.4545418918132782, 0.014711618423461914, -1.450918436050415, -0.9413972496986389, -0.3835086226463318, -0.1110559031367302, -0.16488954424858093, -0.6541319489479065, 0.4968682825565338, -0.4607379138469696, -0.3831747770309448, 2.3226373195648193, -1.3282487392425537, 0.044105466455221176, 2.435445547103882, -0.7553452849388123, -1.655365228652954, -1.437441110610962, 0.38248005509376526, -0.27495166659355164, 1.2271735668182373, 0.405990868806839, 0.7370147109031677, -0.2732369899749756, -0.1198788657784462, 0.34457799792289734, 0.4149656593799591, -0.9803626537322998, -0.8078548908233643, 2.009554862976074, 0.16533935070037842, -2.323991060256958, -0.7841596603393555, 0.3969343602657318, -0.6608436107635498, 1.2091110944747925, -1.9079455137252808, 0.226007878780365, -0.704684853553772, 1.3896024227142334, 1.0786268711090088, 0.6440380811691284, 0.36628103256225586, 1.6437699794769287, -2.0882275104522705, 0.15291716158390045, 0.5144861936569214, 1.8219090700149536, 0.36002084612846375, 1.0763949155807495, 0.6395936012268066, 0.4389919340610504, -0.58163982629776, 0.8902316689491272, -1.585006833076477, 0.10813190788030624, -0.3133467435836792, -2.1370630264282227, 0.056707996875047684, 1.1718637943267822, 0.7322306632995605, 0.5492442846298218, -0.02430884726345539, -0.7173755168914795, -0.5659865140914917, -2.002044200897217, 1.4712169170379639, 0.804969072341919, -0.4911953806877136, 0.10170886665582657, -1.5503100156784058, 1.0726628303527832, -0.6829448342323303, -0.3320455849170685, -2.253458023071289, -1.6758456230163574, 0.1499268263578415, -0.4949462413787842, 0.3911455571651459, 0.5201542377471924, -0.2042209953069687, 0.09780041873455048, 1.343245506286621, 1.9812061786651611, 0.16905754804611206, 0.6206749677658081, -0.21901936829090118, -1.7497279644012451, 0.47258836030960083, -1.066673755645752, -1.3328694105148315, -0.792037308216095, 0.29211440682411194, 1.020511507987976, -0.08469045162200928, -1.03555428981781, -0.8381768465042114, 0.21179993450641632, -0.1882709413766861, -0.2021469622850418, -0.21977613866329193, -0.8156774640083313, -0.6791977882385254, -0.9771034717559814, 0.1900426149368286, -0.6608357429504395, 1.0433712005615234, 1.7966867685317993, -0.13941611349582672, 0.3399154543876648, 0.5650801062583923, -0.6933538317680359, -0.9059010148048401, 0.8732523322105408, 0.6371913552284241, 1.00072181224823, -0.18091754615306854, 0.3825458586215973, -0.6263382434844971, -1.2482578754425049, 0.40811336040496826, -1.526369571685791, -0.31966632604599, 1.0067741870880127, -0.657130777835846, 1.6862503290176392, -1.643067717552185, -0.26368463039398193, 1.0236570835113525, 0.7911830544471741, 0.24018742144107819, -0.026125626638531685, -0.22288423776626587, 0.023537233471870422, -1.6826965808868408, 0.01905490644276142, -1.517408013343811, 0.3547455966472626, -0.2940763533115387, -1.7362257242202759, 2.5194509029388428, 0.25066617131233215, -0.4903227984905243, 0.5333260297775269, 0.5514765977859497, 0.3260079622268677, 1.1533957719802856, 0.17204663157463074, -0.8401803970336914, 1.2029796838760376, -2.112583875656128, 0.028101423755288124, -0.213910311460495, -1.1777863502502441, -0.6053476333618164, -0.16433387994766235, 1.8149083852767944, 0.5387120842933655, -0.5649818181991577, -1.274820327758789, -1.2932345867156982, 0.6618387699127197, 1.213690161705017, -0.4877850115299225, -0.0036227311939001083, 0.9573068022727966, 0.9775368571281433, 0.8590018153190613, 0.19726839661598206, -0.4337151050567627, -0.17449833452701569, -0.3872487246990204, -1.1181426048278809, 0.8419092893600464, 1.968459129333496, 0.6181021332740784, 0.5616270899772644, 0.9401994347572327, -0.014009521342813969, -0.7574827671051025, 0.6141722202301025, 0.4097226858139038, -1.4108316898345947, 1.049096703529358, -0.7918463349342346, 0.9244181513786316, 0.11809256672859192, -0.5093831419944763, -0.9023868441581726, -0.8913450241088867, -1.903050422668457, -0.9664616584777832, 0.14316031336784363, -1.3665956258773804, -0.42221447825431824, 0.4133603274822235, -2.4276695251464844, -1.1392827033996582, 1.040285587310791, -0.6485775113105774, -0.38399040699005127, 0.8668878078460693, -1.1804097890853882, -2.6991631984710693, 0.6362320780754089, 0.27453678846359253, -1.0568989515304565, -2.4216535091400146, 1.4366099834442139, -0.1294407695531845, 0.0497627928853035, 0.6023318767547607, 1.172268271446228, -0.4232030510902405, -0.0400414876639843, 1.7341089248657227, -0.05673541501164436, 0.06340647488832474, 0.9496986865997314, -2.5355048179626465, 0.2659797966480255, -0.9683730006217957, 0.08427606523036957, -0.29010921716690063, -0.48609036207199097, 0.6778541207313538, -0.531997561454773, -2.32395601272583, 1.6361981630325317, -0.6494061350822449, -0.23916758596897125, 0.9588046073913574, 0.5220412015914917, -0.5864295363426208, -0.3439117670059204, 0.6989185214042664, -0.5962659120559692, 0.49474406242370605, 2.6163265705108643, -0.370500773191452, -2.428194284439087, -1.3609024286270142, 0.6181468367576599, -1.274989128112793, 0.5614119172096252, 0.6585406064987183, 1.5771280527114868, -1.0982526540756226, 1.7431774139404297, -1.529363751411438, 0.6275295615196228, 1.2844804525375366, 0.8070903420448303, 0.3111852705478668, 0.24605663120746613, -0.24233746528625488, -0.16226300597190857, 0.5288872122764587, 0.5798529386520386, 0.1177249625325203, -0.34869253635406494, -0.5765216946601868, 0.005596677772700787, 0.3794407844543457, 0.9692389965057373, -0.853350818157196, 0.028652789071202278, -1.0704807043075562, 0.9942770600318909, 0.3687689006328583, -0.976706862449646, -1.9047307968139648, 0.9027083516120911, -0.9113925099372864, -0.6410226821899414, 1.7332744598388672, -0.5514804720878601, -0.4371192455291748, 1.036794662475586, 0.2989693284034729, 1.1333638429641724, -0.6532132029533386, -0.6253604888916016, -0.7654910087585449, 1.8821603059768677, -0.8932624459266663, -0.4585098922252655, 0.008837336674332619, 2.499718189239502, 0.2828422486782074, 0.24116986989974976, -1.7630915641784668, 0.3010311722755432, -0.43469613790512085, -0.7055643796920776, -0.26415327191352844, 1.8949826955795288, -1.102344036102295, 0.09735990315675735, 1.2725530862808228, -0.703076183795929, -1.9225471019744873, 0.6378545761108398, -0.4862716495990753, -0.7299171686172485, 1.5057034492492676, 0.07961864024400711, 1.1416279077529907, -0.428569495677948, -0.9316014647483826, 1.597256064414978, 1.4928200244903564, -1.0261499881744385, -0.8135435581207275, 0.3844607174396515, -0.6162068843841553, 1.3017735481262207, -0.8635827898979187, -0.9880411028862, -1.772209644317627, -0.5827480554580688, -0.8257375955581665, 0.013358662836253643, 0.8009006381034851, -0.08026918023824692, 2.3974599838256836, -0.8156939148902893, -0.442345529794693, 0.3106784224510193, 0.24211162328720093, 1.5151864290237427, 1.0618388652801514, 1.2466981410980225, -0.3765254318714142, 0.5388237833976746, 0.402908056974411, -0.17082034051418304, -1.3185019493103027, 0.05728363245725632, 1.0532276630401611, -0.7531765699386597, -0.17403605580329895, 0.2536143362522125, -0.3392748236656189, 0.8563727736473083, 1.4818000793457031, -0.6996344327926636, -0.6509300470352173, -2.4715373516082764, -1.005864143371582, -0.02034689672291279, -0.8986450433731079, 0.9400944709777832, -0.5382935404777527, -0.42435401678085327, 0.05300808697938919, 1.5722119808197021, 1.1752506494522095, 0.08627625554800034, 0.027432719245553017, 1.0176563262939453, 0.9600756764411926, 0.022916346788406372, 0.2118155062198639, -1.2169520854949951, 0.18749584257602692, -0.8038721084594727, 1.864759922027588, -0.08729139715433121, 0.3213406801223755, 0.6381263732910156, 1.542862057685852, -1.0612622499465942, 1.194148302078247, -1.6977143287658691, 0.36160531640052795, 0.8361576795578003, -1.4900370836257935, -0.24896417558193207, -1.3913226127624512, -0.6614759564399719, 0.2933118939399719, 0.8901253342628479, -0.4770892262458801, -0.1438378542661667, -1.1844538450241089, 0.5382644534111023, -0.4662860929965973, 0.10066282749176025, 0.5513992309570312, -1.6709558963775635, -0.7584213018417358, -1.4042857885360718, 1.2144373655319214, 1.0104904174804688, -2.0841267108917236, 0.7231981754302979, 1.2066450119018555, -0.27090564370155334, -2.70102858543396, 1.0958466529846191, 0.8746693730354309, -0.5805957913398743, -0.7620220184326172, 1.2109323740005493, 0.20585905015468597, -0.19695641100406647, 0.2157033532857895, -1.8431984186172485, 1.1777973175048828, 0.6944831013679504, 0.6685329675674438, -0.7472633123397827, -0.5205204486846924, 0.08632581681013107, -0.3022903800010681, 0.7810683846473694, 0.05616603419184685, -1.0604732036590576, -0.8047637343406677, 0.23876214027404785, -0.704817533493042, -2.1479039192199707, 0.437939316034317, 0.6097338199615479, -1.9104583263397217, 0.3654819428920746, -0.9931632280349731, -1.325021505355835, 0.6587367057800293, -0.693535327911377, 0.2627487778663635, -1.0312682390213013, -0.1892162710428238, 0.3099629580974579, 1.167224407196045, -0.6292112469673157, 2.372995376586914, 0.3607313930988312, 0.7044907212257385, 1.4435501098632812, 1.5431404113769531, 0.12888562679290771, 0.6275327205657959, 0.21398061513900757, -0.022275026887655258, -0.32372787594795227, -0.6497630476951599, -1.448175072669983, -0.5592217445373535, -0.3353319466114044, 0.5391497015953064, -0.24169112741947174, 1.1231350898742676, 0.7698197960853577, -0.9826779961585999, -0.6667818427085876, 0.7018977403640747, -1.3177913427352905, -0.6441076397895813, -1.0073879957199097, 1.366982340812683, -0.17378835380077362, -0.6223303079605103, 0.10943055897951126, 1.6529115438461304, -1.5274139642715454, 1.2256767749786377, 0.7323312759399414, 0.2069614678621292, -0.6268166899681091, 1.0838770866394043, -1.30106520652771, -0.4375074505805969, 0.9454474449157715, -0.7385927438735962, -0.36574387550354004, 0.9505068063735962, 1.6599632501602173, -0.6443474292755127, 0.42756348848342896, -0.1485641598701477, 0.8049112558364868, -0.431856632232666, 1.6520352363586426, 0.6202669739723206, -0.17503902316093445, 1.7637012004852295, -0.9477916955947876, -0.38769975304603577, 2.250349998474121, -1.3377636671066284, 0.2497912496328354, -0.17194636166095734, -0.3791312575340271, -2.2066330909729004, -0.9332894682884216, -1.6220088005065918, -1.7179625034332275, -2.3280189037323, -0.3419792652130127, 0.6106951832771301, 0.2112167775630951, -0.9995719194412231, -1.2879829406738281, -0.3788527250289917, 1.934456706047058, -0.22445322573184967, -0.5522451996803284, -0.02005827985703945, 0.14495475590229034, 0.21254666149616241, -1.410226583480835, -1.4181807041168213, -0.23998315632343292, 0.924734354019165, 0.25012847781181335, 0.2934882342815399, 1.7129400968551636, -1.3578754663467407, 0.4659017324447632, -0.14325891435146332, 1.083686113357544, -0.28654903173446655, -0.09095079451799393, 1.2225888967514038, 0.1432407796382904, -0.44849467277526855, -1.1524810791015625, -1.2054039239883423, 1.3700512647628784, 1.2041432857513428, -0.6585265398025513, 1.7231720685958862, 0.1343955099582672, -0.1655004620552063, 1.6127492189407349, 0.5558037757873535, -1.592374563217163, -0.3665272295475006, -0.4579731822013855, -1.513256311416626, 1.6852481365203857, -0.6779320240020752, 0.5580978989601135, -0.833055317401886, 0.6007510423660278, 1.703766107559204, -0.005395069252699614, 0.7613546848297119, 1.0616339445114136, -1.3194105625152588, 1.012704849243164, -0.2422015368938446, -1.0682545900344849, -0.09739982336759567, 1.0618221759796143, -1.3834636211395264, -0.6894869208335876, 0.4552314877510071, 0.14480991661548615, 1.8304641246795654, -1.6213791370391846, 1.5674045085906982, -1.345879316329956, 0.23592978715896606, 0.9324973225593567, 0.9762954115867615, -0.5761879086494446, -0.2908301055431366, 1.1018325090408325, 0.15340882539749146, -0.12920774519443512, 1.0670721530914307, -1.4185081720352173, -1.5507601499557495, 0.03980787470936775, -0.9643332958221436, 0.5496811270713806, 0.9156278967857361, -1.3666294813156128, -0.7791489362716675, 0.1696540117263794, -0.2372729629278183, 0.925030529499054, 0.7434568405151367, 0.2630643844604492, 0.2799729108810425, -2.137009382247925, 0.5872085094451904, 0.2755489647388458, 0.42843660712242126, 0.22810006141662598, -0.19606229662895203, -0.4392046630382538, -0.8619016408920288, 0.8747067451477051, -0.029419943690299988, -0.21813051402568817, 0.6031129956245422, -0.2800516188144684, 0.45208489894866943, -0.9866793155670166, -0.8208081722259521, -0.7000563144683838, 0.10165884345769882, -0.9101983308792114, -1.6514205932617188, -0.6486722230911255, -0.90534508228302, 1.1445016860961914, 0.025920992717146873, -0.44034484028816223, -0.189665749669075, -0.570202648639679, 1.387877106666565, -1.7432340383529663, 0.11007140576839447, -1.0773639678955078, -0.2336946278810501, 0.9129096865653992, 0.39544299244880676, 0.3559601902961731, 1.756888508796692, -0.6025413870811462, 1.1600464582443237, -0.7071794271469116, -1.0379997491836548, 0.12225554138422012, -0.000785421347245574, -1.5060704946517944, 0.8578230738639832, -0.8295950293540955, 0.16138048470020294, 0.021440068259835243, -0.48025810718536377, 0.6254069805145264, 1.7618316411972046, 1.038388729095459, 1.094940423965454, -2.3840525150299072, -0.2625579833984375, -0.8571861386299133, 0.6879544854164124, -0.13559140264987946, 0.2827167212963104, 0.9777677059173584, 1.1965123414993286, 0.06448899954557419, -1.0274357795715332, -1.2353556156158447, -0.04222206026315689, -1.523730754852295, 2.5061028003692627, 0.261909544467926, -0.46798887848854065, 1.4546983242034912, 0.5009410381317139, 1.4010034799575806, -1.763049840927124, -0.5816553235054016, -1.559038758277893, -1.7035857439041138, -1.6760718822479248, -0.1600153148174286, -1.642688512802124, -0.2743339538574219, -0.41242533922195435, 1.2957454919815063, 0.9810296893119812, -0.5079570412635803, -0.08664459735155106, 1.2019548416137695, -2.0284922122955322, 2.0053813457489014, -0.8971356153488159, -2.8104357719421387, 0.4715564250946045, -0.14827218651771545, -0.2339976280927658, 1.5088509321212769, -1.0043895244598389, 0.9197815656661987, 0.4382227957248688, 0.848770022392273, -0.3432616591453552, 0.27218955755233765, -0.10195630043745041, 0.978747546672821, -0.3065640330314636, 0.6462984085083008, -0.12809737026691437, -1.1012792587280273, 1.3538285493850708, -1.3123341798782349, -1.2593199014663696, 0.31396254897117615, 0.452392578125, 0.13160833716392517, 1.29840886592865, -0.650629997253418, 1.0621761083602905, -1.5350613594055176, 0.6714267134666443, -2.7964582443237305, 1.3104718923568726, 0.3497183620929718, -0.7670484185218811, 0.746568500995636, 1.354693055152893, 1.5068844556808472, -0.5439556241035461, 0.024448363110423088, -1.1612331867218018, 0.3160669505596161, -1.2209036350250244, -0.1258445680141449, 2.839808702468872, -0.11862420290708542, 0.45292380452156067, -0.0474034920334816, -0.04342251271009445, 1.1686733961105347, -0.7625150084495544, -0.12073778361082077, -1.5528627634048462, 0.597678542137146, -0.8088628649711609, -0.9302616715431213, -0.36238420009613037, 1.2400317192077637, 0.8411188721656799, -0.3454612195491791, -0.3978533446788788, -0.8275464177131653, 0.4456816017627716, 0.36964303255081177, -0.5689741373062134, 1.2623223066329956, 1.0107730627059937, 1.2678941488265991, -1.1696912050247192, -0.891798198223114, 0.4569692015647888, 0.22993645071983337, -0.39663365483283997, 0.7415580153465271, 0.18140342831611633, -0.26623526215553284, -0.47042325139045715, 0.7102846503257751, -2.2154414653778076, -2.3528964519500732, 1.0774216651916504, -0.6241108179092407, -1.1987875699996948, 0.4390656352043152, -0.9071283936500549, 1.3655236959457397, -0.1894955188035965, -0.20692960917949677, -0.6071268320083618, -0.4201209247112274, -0.5960983037948608, 0.47186213731765747, -0.6229886412620544, -1.6977800130844116, 0.1605379581451416, -0.6006684899330139, 0.8500820994377136, -1.1715641021728516, 0.28844153881073, 0.7321777939796448, -0.323184996843338, -1.312855839729309, -0.9112293124198914, 1.367443561553955, 2.555400848388672, -0.16830086708068848, -1.7447798252105713, -0.06331715732812881, -1.2242716550827026, 0.5198675394058228, 0.5558537244796753, 0.21567551791667938, 0.6654741764068604, 0.10518667846918106, 1.3635534048080444, -1.8858908414840698, -0.28102827072143555, 1.24319326877594, 0.10196707397699356, -0.22686344385147095, -0.47865113615989685, 0.4320094585418701, -1.5183414220809937, -0.08126017451286316, -0.0057161240838468075, 0.21251675486564636, -1.8031977415084839, 0.6582239866256714, -1.2214360237121582, 1.4089871644973755, -0.048932913690805435, -1.6157042980194092, -0.2460009902715683, -0.7074852585792542, 0.7368583679199219, 1.455844759941101, 1.0380610227584839, -1.2563899755477905, 0.21509014070034027, -1.2273598909378052, -0.1125255674123764, 0.6156176924705505, -1.9845433235168457, 0.5685552954673767, -1.1875723600387573, 1.5254297256469727, 2.382331132888794, 1.9033464193344116, -0.7887043356895447, -0.6367846727371216, -0.2734595239162445, -0.9635707139968872, -1.6040980815887451, -0.2303629219532013, -0.2969920039176941, 0.5713731646537781, 0.5595067739486694, -0.20730073750019073, -0.6824341416358948, 0.21476218104362488, -1.2031925916671753, 0.6125459671020508, -1.5316866636276245, 0.22290684282779694, -0.4367538094520569, -0.752007007598877, -0.21138307452201843, 0.1411081850528717, 0.828644335269928, 1.1317692995071411, -1.3830405473709106, -0.9074162840843201, 1.5582722425460815, 0.8774210810661316, 0.39063647389411926, -0.15724827349185944, -1.3372670412063599, 1.260864019393921, 1.432470440864563, -0.4807504415512085, -0.9381535649299622, 0.4344381093978882, -1.1014842987060547, -0.33145126700401306, 0.04924003779888153, 0.05904185026884079, 1.6929908990859985, -1.3215773105621338, 1.1173839569091797, 1.0877559185028076, -0.9973536729812622, -1.1020129919052124, 0.09532126784324646, -0.29441750049591064, -0.41337552666664124, -0.893032431602478, -1.9340758323669434, 0.22322168946266174, 1.526160717010498, -1.476102590560913, -0.37803032994270325, -1.004107117652893, 0.31227344274520874, 2.034775733947754, -0.1909460574388504, -0.2471943348646164, -0.015204107388854027, -1.3413997888565063, 0.7490084767341614, -2.4365224838256836, 1.1352652311325073, -0.11428038775920868, -0.4920634925365448, -1.7957029342651367, -1.3525254726409912, 0.5999270677566528, -0.8671466112136841, 1.0425478219985962, -0.3232939541339874, -0.4314671456813812, -2.163900375366211, 1.766030192375183, -0.7092545032501221, 0.8596162796020508, -0.82317054271698, -1.6147569417953491, 1.2143586874008179, -0.17058472335338593, 3.3531386852264404, 0.8954993486404419, 0.0007619601092301309, 0.31975477933883667, 1.3615436553955078, 0.02999843657016754, 1.9674055576324463, 0.031099967658519745, 0.05548151955008507, 0.48858746886253357, -1.522793173789978, 0.09257598221302032, -0.43476107716560364, -0.5109245777130127, 0.006520203780382872, 1.168776035308838, -0.3044481575489044, -0.4113316833972931, 0.5911784172058105, -1.071894645690918, 0.24341614544391632, -0.591530978679657, 0.010935870930552483, -0.2149670571088791, -0.5771381855010986, -1.8493016958236694, 0.5577592849731445, 0.9294430613517761, -2.4626736640930176, -1.9056239128112793, -0.4416152834892273, 1.930387258529663, 0.32338789105415344, 0.1476789265871048, -0.29210540652275085, 1.5242336988449097, -0.46574392914772034, -0.14589886367321014, -0.3171641230583191, -1.3452627658843994, -1.1017142534255981, -0.33315616846084595, -0.0009221048094332218, -0.08046133071184158, 0.7493621110916138, -0.38257870078086853, -0.6399469971656799, 1.0584739446640015, -0.7758482694625854, 0.6472173929214478, -0.3309381902217865, -1.839665174484253, -0.5503853559494019, -0.7017503380775452, -0.8729912638664246, 0.15117725729942322, 0.18874609470367432, -1.1387325525283813, -0.2109207808971405, 0.7139601111412048, -0.2573179304599762, 1.9603270292282104, 0.18204937875270844, 0.9593632817268372, -0.36557766795158386, 0.7874550819396973, -0.6190552711486816, 1.0085610151290894, 0.16131526231765747, -0.5956258177757263, -1.2099010944366455, 0.3031919598579407, 2.2687056064605713, 0.18778535723686218, 0.2882310748100281, -1.7095022201538086, 1.2014837265014648, 1.1793240308761597, 0.44857484102249146, -2.422266721725464, -1.5700523853302002, 0.4297594726085663, -1.3501152992248535, -1.4450809955596924, -0.7315094470977783, -0.9859859943389893, 1.8687664270401, -0.26106759905815125, 0.4210759997367859, -1.144247055053711, -0.7537699937820435, -0.6504496932029724, -1.30643630027771, -1.361553430557251, 0.36261051893234253, -1.1883023977279663, -0.20263396203517914, 0.48397740721702576, -0.40977486968040466, 0.885354220867157, -0.4395338296890259, -1.327336311340332, 0.35165834426879883, 0.7249476313591003, 1.8095085620880127, -0.8547351360321045, 2.309572458267212, 0.9243065714836121, 1.0285725593566895, 0.22943830490112305, 1.1603996753692627, -0.989368736743927, 0.12622058391571045, 2.532010555267334, 0.7551933526992798, 0.11205396056175232, 0.6134999990463257, -1.585686206817627, 0.05637392774224281, 0.4941326677799225, -0.6918350458145142, -1.6433359384536743, -0.6263025403022766, 0.5684733390808105, -1.1068141460418701, 1.0862843990325928, 0.5607285499572754, -1.5347108840942383, -0.22588448226451874, 1.1382253170013428, 1.5953062772750854, -1.8811893463134766, -0.053241629153490067, 0.22025175392627716, -0.7121816873550415, 0.3971962332725525, 1.6181162595748901, 0.17951180040836334, 0.5797513127326965, -0.08845586329698563, 0.8915709257125854, 0.5777825117111206, 2.095716953277588, -0.7285788655281067, 0.23073533177375793, -2.300969362258911, 1.0279083251953125, 1.046800971031189, -0.3785754442214966, 1.0846450328826904, 0.7527322173118591, -1.4943360090255737, 1.0468369722366333, -0.01654641516506672, 1.023036003112793, -0.061628568917512894, 0.8143460750579834, -0.8836953043937683, -0.12626175582408905, 0.6196545958518982, -1.317644476890564, 2.0148351192474365, 1.2071754932403564, -0.6914364099502563, 0.8661720156669617, -0.6636070013046265, 0.011072761379182339, -0.9405165910720825, -0.31565338373184204, 0.8307303786277771, 0.4410339295864105, -0.22345779836177826, -1.538689136505127, 0.7347729206085205, -1.1233665943145752, 1.4574995040893555, -1.6899783611297607, -0.6495575308799744, -0.5263136029243469, -1.1832516193389893, 0.567382276058197, -1.3199762105941772, 0.4871930480003357, 0.17561876773834229, 0.9283429980278015, 0.00960139837116003, 2.0221304893493652, 0.24279876053333282, -0.16730231046676636, -0.311835914850235, 1.1325128078460693, -0.22603876888751984, -1.1387017965316772, 1.2519818544387817, 0.04771079868078232, 0.11609448492527008, 0.43702101707458496, -0.34827950596809387, -0.1396782249212265, 0.4460000991821289, -1.0632966756820679, -0.41291776299476624, -0.4503399133682251, 0.023083463311195374, -0.35044991970062256, 0.2534661293029785, 0.32791247963905334, 1.5256321430206299, 0.24383383989334106, -1.6999306678771973, -0.5237017869949341, -0.4012232720851898, -1.1007462739944458, -0.7094013690948486, -0.04000081121921539, 2.0606648921966553, -2.32594895362854, -1.2574183940887451, -0.9533668160438538, 0.6894271373748779, 0.42875757813453674, 2.155348300933838, -0.9353736042976379, 0.7686855792999268, 0.38335931301116943, -0.19634635746479034, 0.26948922872543335, -0.1677856147289276, -0.19918806850910187, -1.4340218305587769, -1.184988260269165, -0.4312400817871094, 0.07393438369035721, 1.3449043035507202, 0.46866828203201294, 1.3577101230621338, 0.18226666748523712, -1.336702823638916, 0.0302545465528965, -1.063674807548523, -0.10376645624637604, -0.7455017566680908, 0.11948709189891815, 0.0015615380834788084, -0.42191532254219055, -0.7301589250564575, 0.7814456224441528, 0.15702146291732788, -1.2061996459960938, 0.5286745429039001, -0.3097052276134491, 0.2539532780647278, -1.6512243747711182, 1.82895028591156, 0.5083967447280884, -0.4463891088962555, -0.8888219594955444, 0.474899560213089, -1.1930935382843018, 0.49287840723991394, -0.21900296211242676, 0.06888863444328308, -0.4259681701660156, -1.2362829446792603, 1.5914195775985718, 0.6700506210327148, 0.722062885761261, 0.406728595495224, -1.3745464086532593, 1.2724488973617554, -0.8443562984466553, 0.9120364189147949, 0.32802265882492065, -1.3867034912109375, -0.947311282157898, 1.6941357851028442, 0.6496354937553406, 1.262302279472351, -1.232532262802124, 0.5197590589523315, -0.7706721425056458, 1.3459912538528442, 0.31342703104019165, -1.700045108795166, -1.1908591985702515, 0.5096976161003113, -0.1441536396741867, 0.868044912815094, 0.5900309085845947, 0.30980128049850464, 0.4875049293041229, 0.5553705096244812, 1.4339675903320312, 1.016174077987671, -2.0378153324127197, 0.3233692944049835, -0.4635799527168274, 0.13679565489292145, -0.3650689125061035, 1.3341909646987915, 0.7922961115837097, 1.407866358757019, -0.16133737564086914, -1.5099765062332153, -1.3766956329345703, 1.6137880086898804, -1.4363535642623901, 0.07349893450737, -0.08690255880355835, -1.1455055475234985, 0.27756381034851074, 0.18581148982048035, 0.5793554186820984, -1.1606354713439941, 0.1355305314064026, -0.8945244550704956, -0.6090392470359802, -0.47764062881469727, 0.7422788143157959, -0.17123636603355408, -0.7254661917686462, -1.4525713920593262, -0.4461236298084259, 0.7686114311218262, -0.32668209075927734, 1.3364636898040771, -0.27394717931747437, 1.6291207075119019, -0.9565063118934631, -0.7807899117469788, -0.5290272235870361, -0.1910489946603775, -0.48940011858940125, 1.4410828351974487, -1.2205172777175903, 0.1389644891023636, -1.2585535049438477, -0.6189808249473572, -1.7975335121154785, -1.1471962928771973, 2.7522642612457275, -0.5080661773681641, -0.719571590423584, 0.2555966377258301, 1.0207775831222534, 0.162501260638237, -0.33820104598999023, -1.326545000076294, -0.06951692700386047, -0.38346830010414124, 0.3921016454696655, 1.1527745723724365, 0.25859972834587097, 0.7093296051025391, -0.13923002779483795, -0.2976926565170288, 2.0273361206054688, -0.2810687720775604, 0.7042567729949951, 0.590411365032196, -1.5987896919250488, 0.6298527121543884, -0.4651659429073334, 0.337399423122406, -0.42491212487220764, -0.9614322185516357, 1.219927430152893, 0.6066650748252869, 0.24985817074775696, 1.350417971611023, 0.31482934951782227, 0.18684835731983185, -1.0546793937683105, 0.7303094267845154, -0.8820402026176453, -1.108373999595642, 0.34806346893310547, 0.8015264868736267, -1.4736329317092896, 0.9439727067947388, -0.5163148641586304, -0.8308424353599548, 0.47836941480636597, -0.02377162128686905, 0.20445013046264648, 0.12951722741127014, -0.8203513622283936, 0.2988884449005127, 1.2551825046539307, -0.8325682878494263, -0.6557872891426086, 0.39901605248451233, 1.524485468864441, 0.15647006034851074, -1.3930318355560303, 1.2662522792816162, 0.05347612500190735, -1.223249912261963, 0.26060032844543457, 2.0322389602661133, -0.13119271397590637, 0.27057212591171265, -0.04169309139251709, 0.316613107919693, 0.3577384054660797, -0.11444009095430374, 0.09888532757759094, -0.009849267080426216, 0.517479419708252, 0.563814640045166, 0.6436415910720825, -0.8337996602058411, -0.46018552780151367, 0.5875693559646606, -0.7099701762199402, 0.7100518941879272, 0.47873732447624207, -1.0216326713562012, 1.5918898582458496, 1.4045491218566895, -1.3463976383209229, -1.2161686420440674, 0.036070067435503006, 1.1588380336761475, 0.7778631448745728, -1.2984153032302856, -0.6225640773773193, -0.8093651533126831, -0.7619825005531311, -1.9837138652801514, 1.8454344272613525, 1.8747923374176025, 0.33562523126602173, -0.3781406879425049, -0.1864032745361328, 1.262261152267456, 0.6892878413200378, 1.50649893283844, 0.09236022084951401, -0.4735472500324249, -0.1408267766237259, 1.4726476669311523, 0.1806240826845169, 1.0509750843048096, 0.6522712707519531, -0.42446407675743103, -0.2925606966018677, -0.8228764533996582, 2.118410110473633, -1.3246403932571411, -0.3305976390838623, 1.3794941902160645, 1.954062581062317, 0.5070140957832336, -1.5990931987762451, 1.3365145921707153, 0.6604741811752319, 3.5024242401123047, -0.6081374287605286, -0.8870882391929626, -0.9497885704040527, -0.04359735548496246, 0.41520047187805176, -0.08100436627864838, -0.3650067150592804, -0.8779522776603699, 0.7676517963409424, 0.3744323253631592, -0.13089635968208313, -1.8018637895584106, 0.06301480531692505, -0.32646527886390686, 0.4681805968284607, -0.5717433094978333, 0.9604518413543701, 1.7636489868164062, -0.23732639849185944, -0.18724766373634338, -1.7674716711044312, 0.05574502423405647, 0.6932138204574585, -0.8975796103477478, 2.394580364227295, -0.3535109758377075, 0.20915132761001587, -1.5340181589126587, -1.467332363128662, 0.6016090512275696, 0.01829744502902031, -0.23971065878868103, -0.71776282787323, -0.689074695110321, 0.19375881552696228, 0.4791216254234314, -0.3738252520561218, -1.2810777425765991, 1.4669915437698364, -0.09928736835718155, 1.075835108757019, 0.9171529412269592, -1.545522689819336, -0.9000557661056519, 0.5673004984855652, -0.14162613451480865, -0.5262667536735535, 0.3335745930671692, 0.3593713641166687, -2.1965949535369873, 0.1501857191324234, 1.8294273614883423, -0.0428292378783226, 0.9528617262840271, 1.2315865755081177, 0.9305512309074402, -0.1370926797389984, -0.1987960785627365, -0.19481036067008972, -1.3089195489883423, 1.8414949178695679, -0.6405401229858398, -0.6508299708366394, 0.426533967256546, 0.10474235564470291, -1.313547134399414, -0.577290415763855, -0.7494249939918518, 0.39800336956977844, 0.4505476951599121, 1.9175310134887695, 0.9818801283836365, 0.1884101778268814, -0.9543702602386475, -1.1338297128677368, 0.7535797953605652, -0.25349724292755127, -0.4440612196922302, 0.8727179765701294, 1.5876890420913696, -1.6597974300384521, -0.6830152869224548, 0.33133360743522644, -1.0483499765396118, -0.7772534489631653, -0.03279615193605423, 0.6397263407707214, 1.2884787321090698, -2.860450029373169, 1.1710941791534424, -0.8248758316040039, 0.9083877801895142, 0.38081175088882446, 0.8980482816696167, -0.06928938627243042, 1.4322118759155273, 0.3302742540836334, -0.6558980941772461, 0.631356954574585, -0.4673953056335449, -0.40678057074546814, -0.29622921347618103, 0.42093679308891296, -0.615532636642456, -0.30668386816978455, -0.25959479808807373, 0.5580736994743347, 0.948425829410553, 1.061976671218872, -0.5638312697410583, 0.3202393054962158, 0.8175445795059204, -0.6221558451652527, 1.1779850721359253, 1.0478811264038086, -0.5530981421470642, 0.8152152895927429, -0.33476606011390686, 2.1873626708984375, -0.5886641144752502, 1.2702183723449707, 2.3233773708343506, 0.29877564311027527, 0.3125205934047699, 0.275481253862381, 0.3811227083206177, 1.9147173166275024, 0.15397076308727264, -0.8345257639884949, 0.014837049879133701, 0.6028117537498474, -1.5019136667251587, -1.0539536476135254, -0.9420920014381409, -0.3281833827495575, 0.05206052586436272, -0.5742306113243103, -0.12170454859733582, 1.3890388011932373, 1.2934757471084595, 1.8626408576965332, 1.114414095878601, -0.5521644949913025, 0.49769535660743713, -2.3456621170043945, -0.3054840564727783, -0.45899057388305664, -1.7366836071014404, 1.0717408657073975, 0.7827457785606384, -1.316391110420227, 0.5158695578575134, 0.5535021424293518, 0.3885771334171295, 0.1407884955406189, -0.8046122789382935, -1.3275845050811768, 0.5814349055290222, 0.453959584236145, 0.9191249012947083, -1.062507152557373, -0.332780659198761, -1.59756600856781, 0.8436193466186523, -0.997962236404419, -0.31261399388313293, -0.9868370890617371, 0.7190290689468384, 1.1891473531723022, 0.6595538258552551, 0.5953256487846375, -0.48285725712776184, -0.6550331115722656, 0.1553470641374588, -0.2249608337879181, 0.48157620429992676, -0.1626298874616623, -0.01808493211865425, -0.0672295093536377, -1.0705865621566772, 0.9513555765151978, 0.869045615196228, 0.70192950963974, 0.4324667751789093, -0.42107903957366943, -0.4407397508621216, -1.2019678354263306, 0.8494858741760254, 0.18233169615268707, -1.3018146753311157, -0.8838509321212769, 1.0055835247039795, 0.620995283126831, 0.9517261385917664, 0.06008046865463257, 1.2715024948120117, -0.8905842304229736, 0.5803792476654053, 1.9749411344528198, 1.2097418308258057, 0.11354319006204605, 2.180168628692627, -0.48538053035736084, -1.5183336734771729, -0.7012749910354614, 0.24214550852775574, 0.30618810653686523, -0.9967050552368164, 0.14475592970848083, -0.14339841902256012, -0.02365740016102791, -0.4365933835506439, 0.22544442117214203, 0.13796062767505646, 0.9585340023040771, 0.4234388470649719, 1.667677640914917, 0.5013134479522705, 0.3816930055618286, -1.732138991355896, 1.4624123573303223, 1.4561679363250732, -0.20019252598285675, -0.3418976962566376, 0.760918140411377, -0.7023655772209167, 1.2066096067428589, 1.3386908769607544, -0.31020697951316833, -0.8230606317520142, -0.23694302141666412, 0.6233713626861572, 0.41114795207977295, 0.9321771264076233, 1.2060556411743164, -0.16462735831737518, 0.06992117315530777, -1.274967074394226, -0.32460275292396545, 1.5402255058288574, 1.1201250553131104, -0.1462818831205368, -0.9541557431221008, 0.9910022020339966, -0.381777286529541, -0.2083226591348648, -0.3654402792453766, -0.136370450258255, -0.09571307897567749, -1.5859895944595337, -0.6637159585952759, -0.26076051592826843, -0.3796808123588562, -0.5800310969352722, 0.6119077205657959, -0.26574796438217163, 0.9146718978881836, -0.9506006240844727, 1.952301025390625, -0.036196768283843994, -0.2963317334651947, 1.9354175329208374, 0.1355305016040802, -0.9693331122398376, -1.7701293230056763, -0.22585560381412506, -0.259969562292099, 1.3604434728622437, 0.3560228645801544, -0.26883384585380554, 0.9095593094825745, 0.5727076530456543, -0.8640422224998474, -0.5808946490287781, -1.5272808074951172, -0.4080749750137329, -1.381766676902771, -0.7189685106277466, 0.8981721997261047, -0.8822591304779053, 2.1247143745422363, 0.06558535248041153, 1.9113423824310303, 0.17325036227703094, 0.19919951260089874, 0.5332770943641663, 2.3446764945983887, 1.1896864175796509, 0.0748276561498642, -0.893340528011322, -0.21236927807331085, 2.4138691425323486, 0.7185295820236206, 1.8186559677124023, -1.7913895845413208, -0.45004579424858093, 0.769616425037384, 1.242736577987671, 0.17665603756904602, 0.13725610077381134, 0.7721518278121948, -1.9282879829406738, 0.9671167731285095, 0.8002954721450806, -0.25452545285224915, -1.646856427192688, -0.28194180130958557, -0.8249812722206116, 0.1049664318561554, 0.5676740407943726, 0.46966275572776794, -1.8791308403015137, -0.7937951683998108, -0.07850421965122223, 2.6812238693237305, 0.3230225741863251, -0.586638867855072, -0.4400341510772705, -1.62313711643219, -1.0574100017547607, 0.48792263865470886, 0.30933406949043274, 0.8852564096450806, -0.9328939318656921, 0.44461938738822937, -0.23319122195243835, -0.4525068700313568, 1.006119728088379, -1.844401478767395, 2.2907748222351074, 0.6506516933441162, -1.7221232652664185, 0.7586534023284912, 0.49528247117996216, 1.191620945930481, -0.3977106213569641, -0.7796766757965088, -0.5955008268356323, -0.1256050020456314, 0.8907248377799988, -0.10340187698602676, 1.2620066404342651, 0.7267842888832092, -0.282009094953537, 0.23269608616828918, -0.9560911059379578, 0.6859878897666931, -0.20850884914398193, -1.783320665359497, 2.2552988529205322, -0.581061065196991, -1.5873770713806152, -1.3275915384292603, 1.1361863613128662, 0.895135223865509, 0.9623038172721863, 0.4962383806705475, -1.4733281135559082, -1.4302036762237549, -0.11241879314184189, -0.20067818462848663, -1.8098814487457275, -0.6995446085929871, -1.053420066833496, 1.0372170209884644, 0.2631567120552063, -0.2618432641029358, -0.25369739532470703, 0.03733394667506218, -1.6866596937179565, -0.38232263922691345, -2.50653338432312, 1.0562089681625366, 0.6318593621253967, 0.03245066478848457, -0.5114622712135315, 1.2788712978363037, 0.6214922070503235, -0.15029878914356232, -1.269797682762146, -0.0738808810710907, -1.7990964651107788, -0.2145080268383026, -0.6910684108734131, -0.6787385940551758, 0.4722650945186615, -0.3676016926765442, -0.5137544870376587, -0.5921258926391602, -1.0130707025527954, 0.6137964129447937, -0.8602162599563599, -0.45692941546440125, -0.34252694249153137, 0.472385972738266, -1.1997361183166504, -0.5084493160247803, -0.07823730260133743, 0.4591309428215027, -0.05136366933584213, 0.8782227039337158, 0.12763968110084534, 2.008676528930664, 0.8195434212684631, -0.4187052845954895, -0.4058573842048645, -1.879820466041565, -0.015634113922715187, 0.07195921987295151, -0.3320317566394806, -1.5599877834320068, -0.2766880691051483, 0.5941475033760071, -0.6712872385978699, 0.6790531277656555, -0.032641515135765076, 0.39195793867111206, -0.09312528371810913, -0.27194127440452576, -0.11501263082027435, -0.9073165655136108, -2.6270053386688232, 0.7600584030151367, 0.6860728859901428, -0.4907499849796295, -1.516768217086792, 1.03285813331604, -0.14817796647548676, -0.5971338152885437, 1.0203858613967896, 0.04027185216546059, -1.132678747177124, 0.2688237726688385, 0.12214469164609909, 1.2385845184326172], [1.1875170469284058, 0.04511439427733421, -0.21034371852874756, -1.1714023351669312, -0.5459038615226746, 0.3260895311832428, 0.36744388937950134, 0.8663221001625061, -1.8567756414413452, 0.5431037545204163, 0.9413545727729797, -0.5116531848907471, 0.5097512602806091, -1.5503456592559814, -0.32609033584594727, -0.5465570092201233, -0.5336235761642456, -0.09931633621454239, 1.1852353811264038, 0.007913450710475445, 0.70157390832901, -0.22746695578098297, 0.15721292793750763, 1.2680238485336304, -0.0841442197561264, -0.20534862577915192, 0.28727295994758606, -0.06699976325035095, 0.47114887833595276, 0.3866846263408661, -1.3746610879898071, -0.7586658000946045, -0.29176560044288635, 0.8765943050384521, -0.44283273816108704, 0.28300976753234863, -0.745643138885498, -0.8096293807029724, -0.45977240800857544, 1.8306857347488403, 0.2090909332036972, -0.28165769577026367, -2.587977170944214, -1.115497350692749, 2.6235861778259277, 0.532485842704773, -1.0516791343688965, -1.2681783437728882, -1.0720659494400024, 0.4251895844936371, -0.007417577784508467, -0.8079559803009033, 0.6068148016929626, 0.03411407023668289, 0.7708664536476135, 0.21519312262535095, -0.4481232166290283, -0.42492443323135376, -0.23108014464378357, 0.6800122261047363, 0.688932478427887, -0.9220885634422302, -0.9418125748634338, 0.4482969641685486, -2.715641736984253, -0.3646286427974701, 0.27268871665000916, 3.4786384105682373, -0.05581630766391754, -0.21522757411003113, -0.044846776872873306, 1.016990065574646, -3.1712136268615723, 0.3164283335208893, -1.7426785230636597, 1.6825264692306519, -1.9734420776367188, -0.07634592801332474, -1.0798834562301636, 2.341773271560669, 0.5734763145446777, 1.3066351413726807, 1.3070377111434937, 0.45751267671585083, -0.5406126379966736, -0.8601461052894592, 0.6794224381446838, -0.06447818130254745, -0.47701168060302734, -0.06832366436719894, -0.08970684558153152, 0.16845540702342987, -0.434821218252182, 0.5499938130378723, -0.9001472592353821, -0.1107105165719986, 0.35910362005233765, -0.395076185464859, -0.39914339780807495, -1.2372455596923828, -0.08640193939208984, 0.9220385551452637, -0.43316537141799927, -0.1367414891719818, 2.4798789024353027, -1.357791543006897, -1.0655546188354492, -1.0270055532455444, -0.40734660625457764, 0.5811289548873901, 0.0679340735077858, 1.7183253765106201, -0.5822509527206421, -0.5148244500160217, -1.4186928272247314, -0.03122766874730587, 0.050290368497371674, -0.5907392501831055, 0.43132150173187256, -1.8741837739944458, 0.6057078242301941, 1.3729774951934814, -0.051498621702194214, -0.2130453884601593, 0.09148313850164413, 0.8402425646781921, 0.684077262878418, -1.228832721710205, -0.5958269238471985, 1.4499324560165405, -0.5666952133178711, 0.6420334577560425, 0.2929675579071045, -0.2866418659687042, 0.5432265996932983, -0.9472718238830566, -0.8307638764381409, 0.13159622251987457, -0.5018054246902466, -2.2847533226013184, -0.7913803458213806, 0.03065858595073223, -2.0319082736968994, -0.45803016424179077, -2.367488384246826, -1.2145568132400513, 0.1447466015815735, 0.7929388880729675, 2.054673433303833, -1.3339868783950806, 0.5593554973602295, -0.8356783986091614, -2.5221269130706787, 0.17843930423259735, 1.2548855543136597, 0.707916259765625, 0.6509279012680054, 1.0535995960235596, -1.2023342847824097, 1.4917389154434204, 0.9429915547370911, 0.48609626293182373, 0.6819623708724976, 0.521976113319397, -0.6455932259559631, -0.21801577508449554, -0.20067918300628662, 2.196666955947876, 2.2393009662628174, 1.3771976232528687, -1.1657400131225586, 0.4578436017036438, -1.054973840713501, -0.09271369129419327, -0.7462060451507568, 1.3521610498428345, -1.1442188024520874, -0.1826716810464859, -1.4477125406265259, 1.6002390384674072, -0.4569939076900482, -0.4728010594844818, -1.1687179803848267, 1.7068082094192505, -0.04826787859201431, -0.7073372602462769, 0.7393249273300171, -1.0858234167099, -0.05243930220603943, 0.2958940267562866, 0.8971734642982483, 0.8179014921188354, -0.5198429822921753, -0.33790323138237, -0.8740450739860535, 0.6141513586044312, -0.647153377532959, 0.9257776737213135, -0.8731593489646912, -0.2740163803100586, -1.3303614854812622, -0.016214173287153244, 0.7882731556892395, -0.6217318177223206, 0.7134497761726379, -0.22839373350143433, 0.935054361820221, -0.6393467783927917, -1.89811372756958, 1.9295507669448853, -0.6638102531433105, -0.15551939606666565, 0.13927605748176575, -0.5773974657058716, 2.1530098915100098, -0.033203911036252975, -1.2226439714431763, -1.5916370153427124, 2.083984136581421, -1.1852504014968872, -0.37220504879951477, 1.4683126211166382, -0.9548286199569702, -0.6509631872177124, -0.04930053651332855, 2.1528983116149902, -0.22894684970378876, -1.6329845190048218, -1.8944685459136963, -1.287649393081665, 0.24886447191238403, 0.030578317120671272, -0.04858776554465294, -0.39183470606803894, 0.7545256018638611, -0.23946647346019745, -0.8888281583786011, -1.5569100379943848, -0.4107384979724884, 1.264635443687439, 1.0010337829589844, 0.16749052703380585, 0.47855913639068604, -0.9365962743759155, 1.9052178859710693, -0.07105983793735504, 0.4990719258785248, -0.7683502435684204, -1.27190363407135, -0.9190018177032471, -1.1803234815597534, -0.28237515687942505, -1.9613794088363647, 0.5088592171669006, 0.8306529521942139, -1.2188615798950195, 1.4906587600708008, 0.77720046043396, -0.12693670392036438, -0.4411492645740509, -0.5387264490127563, -0.1617506891489029, -0.6746430993080139, -1.285581350326538, 1.8737452030181885, 0.7291198968887329, -0.5996711850166321, 0.35807004570961, -1.106027603149414, -1.5294021368026733, -0.8048916459083557, 0.67112135887146, -0.8148232102394104, 0.43555912375450134, 1.1488211154937744, 1.108464002609253, -0.22177647054195404, -0.26921623945236206, -0.14800314605236053, 0.22901572287082672, 0.5913932919502258, 2.2864794731140137, 1.4927937984466553, -0.8753877878189087, -1.0200625658035278, 1.3115108013153076, 0.49208515882492065, -0.8959602117538452, -0.03218415006995201, 0.6640786528587341, 0.24826964735984802, -0.48451152443885803, -0.9983406066894531, 0.2843851149082184, 0.9083905816078186, 1.3731732368469238, -0.37923330068588257, 0.2759019136428833, 1.0728493928909302, -0.057183898985385895, 0.14933057129383087, 0.09490583837032318, 0.8638880252838135, 0.7197898030281067, 0.7913418412208557, 0.04686151444911957, 1.5388195514678955, 0.24598079919815063, 0.5326374769210815, -0.7958548665046692, -1.3812837600708008, 0.37415963411331177, -0.5961039066314697, 0.5524083375930786, 0.9116005301475525, 0.2690460681915283, 0.8390846252441406, -0.5082887411117554, -0.15911783277988434, 0.9211868643760681, 0.19634073972702026, 0.6230788826942444, 2.0209832191467285, -0.6330246925354004, 0.9495587944984436, -1.3041467666625977, -0.8121591210365295, 0.2663102447986603, -1.1159915924072266, 0.31306859850883484, 0.40186211466789246, 0.23374006152153015, 0.5144674181938171, -0.3134516179561615, -0.24588562548160553, -0.2406710982322693, -0.03455330431461334, -1.310068964958191, 0.8768495917320251, -1.313398838043213, -0.03672001138329506, 0.770073652267456, 0.03778950124979019, 2.5460078716278076, -2.0174918174743652, 0.08411698043346405, -1.7034235000610352, 0.6550955176353455, 1.986093282699585, -1.2437171936035156, 0.0924704372882843, -0.3051270842552185, -1.4155590534210205, 0.843574583530426, -0.9510531425476074, -0.750986635684967, 1.9990196228027344, 1.0011564493179321, 0.011135352775454521, 1.3990719318389893, -1.2787806987762451, 1.0093662738800049, -0.8775193095207214, -0.9899981021881104, 0.8754404187202454, 1.898416519165039, 0.7634929418563843, 2.5933749675750732, -0.27345049381256104, -2.373927116394043, 0.7168583273887634, 0.11400525271892548, -0.8149219155311584, 0.5794454216957092, -0.42272040247917175, -0.8422999382019043, 0.564443051815033, 0.1348549723625183, -2.2515976428985596, -0.2232782393693924, -0.6932545900344849, -0.9177595376968384, -2.4622390270233154, -0.16159822046756744, -1.0279872417449951, -0.4993736445903778, 0.6148070693016052, 0.5265082120895386, -0.09269847720861435, -0.24265247583389282, -1.8828729391098022, 0.48069053888320923, -0.7788149118423462, -1.2629011869430542, 0.3468184471130371, 0.708251953125, -0.40881800651550293, 1.103688359260559, 0.9085899591445923, 0.8478093147277832, -0.5262372493743896, -0.6844332814216614, -0.47546452283859253, 0.6443585753440857, 1.3551429510116577, 0.5026915669441223, 0.8059049844741821, 0.7010853290557861, -0.5070449709892273, -1.1886152029037476, 1.319239854812622, 0.06524398177862167, 0.6613202095031738, -0.10147619247436523, -1.5540052652359009, -0.5022022128105164, 2.0546913146972656, -0.5907142162322998, 1.466843843460083, 1.4877796173095703, 0.7587429881095886, -1.0446853637695312, 0.2445678859949112, 1.0474908351898193, 0.429978609085083, -1.4006438255310059, -0.9753398895263672, 0.6722481846809387, 0.5008377432823181, -0.1515979915857315, -1.4864630699157715, -1.4676786661148071, 1.3383991718292236, -0.0585777647793293, 2.547593832015991, 0.5992050766944885, -0.11316794902086258, 0.09435071796178818, -0.6613644361495972, 0.17766572535037994, -0.8921661376953125, -1.5932451486587524, -0.3931586742401123, 0.6163027882575989, -0.3474051058292389, -2.302757501602173, 0.012251587584614754, -0.1182003989815712, -0.3993782103061676, 0.8550360798835754, -1.8032726049423218, 1.4831466674804688, -0.5922576189041138, 0.9597288370132446, 0.34918108582496643, -1.425955057144165, 0.9373824000358582, -0.755793571472168, -1.0508185625076294, 0.5349524021148682, -0.010113530792295933, 1.612321138381958, -0.8796916604042053, 0.26532015204429626, -1.024897813796997, -0.32763010263442993, -0.2245800793170929, 0.4902248680591583, -0.6714243292808533, 1.7570576667785645, -0.7069622874259949, 1.0355359315872192, 0.4619901180267334, 0.3038431704044342, -0.25879284739494324, 0.14405497908592224, -0.9540296792984009, 1.112735629081726, -0.5028330087661743, 0.017398536205291748, -0.8294094800949097, 0.07477574050426483, 0.10164434462785721, -0.27095454931259155, 0.6587122678756714, 1.0013368129730225, -0.3917461335659027, -0.3444467782974243, 0.38518333435058594, 0.1656322032213211, -0.8362805247306824, 0.2944425940513611, 0.1440601795911789, -0.8155904412269592, -0.7876186966896057, -0.5562103390693665, 0.7133477926254272, 1.3346399068832397, 0.9857640266418457, -0.1746591031551361, -0.6576116681098938, 0.6732990145683289, -2.055121660232544, 0.9532209038734436, -2.1474082469940186, 1.3148698806762695, -1.1047167778015137, -0.11036112904548645, 1.175176739692688, 0.3884626626968384, 0.21926334500312805, 0.5279417634010315, -0.1719885617494583, 0.5342448353767395, -1.8017597198486328, -0.6211220026016235, 0.08331888169050217, -0.03097454272210598, -0.34642377495765686, -1.137778639793396, -0.9547196626663208, 0.13023705780506134, 0.171584352850914, 1.6328660249710083, -2.4597465991973877, -0.044631022959947586, 0.13657058775424957, -0.8300412893295288, -1.3746192455291748, -1.0666660070419312, 0.15937480330467224, -0.941101610660553, 0.08205464482307434, 0.1084061786532402, -0.6444371342658997, 0.6024569272994995, 0.4279541075229645, 1.4036351442337036, 0.6963233351707458, 1.1693872213363647, 2.342944383621216, 0.4180167317390442, -0.6794946789741516, 2.0386834144592285, 0.6501700282096863, -0.4151575565338135, -1.0942271947860718, -1.5815777778625488, 1.2154711484909058, -0.025922603905200958, -1.370703101158142, 1.1981840133666992, 1.7863787412643433, -2.179755449295044, -0.9440652132034302, 0.9956281185150146, 1.5573687553405762, 0.5772321820259094, 0.7924034595489502, -0.7789638638496399, -0.7666628956794739, 0.1873444765806198, 0.6647763252258301, -0.811466634273529, 0.5941699743270874, -2.1183412075042725, -2.3864033222198486, -1.1550363302230835, -1.2025585174560547, -0.8130192160606384, 0.43173521757125854, -0.4409087002277374, -0.7330858707427979, -2.573603630065918, 1.9733567237854004, 0.6522195935249329, 0.9888368248939514, 1.8956507444381714, 0.8129275441169739, 2.640475034713745, 1.6738882064819336, -2.047750949859619, 0.8501268029212952, -0.6962776780128479, 1.0534507036209106, 0.26522985100746155, -0.7026019096374512, -1.3364624977111816, -0.5038255453109741, -0.24283847212791443, -1.2538989782333374, -0.2772216200828552, -0.7986447215080261, -0.7419463992118835, -1.0744813680648804, -1.4104222059249878, 0.0778106153011322, -0.9886645674705505, 0.09238681197166443, 0.4002966582775116, 1.2452125549316406, 0.818105936050415, 0.8452460765838623, -0.2796013653278351, -0.9060357213020325, -1.2817693948745728, 0.37903594970703125, -1.5964226722717285, -0.04860124737024307, 0.8888980746269226, -0.5082584619522095, -0.3463842570781708, -2.555605411529541, 0.37485262751579285, -0.8402347564697266, -1.1522859334945679, 0.6940789222717285, 0.25330406427383423, 0.4818978011608124, 2.2641375064849854, 0.2736075222492218, -0.5032522678375244, 0.35127896070480347, -1.2254993915557861, 0.2168143391609192, -2.065396308898926, 0.8874328136444092, -0.8582785129547119, 0.6255676746368408, 1.4983853101730347, 0.09751607477664948, -1.180371642112732, 0.5600048899650574, -0.2754778563976288, -1.0277047157287598, -0.775352418422699, 0.8564910888671875, 0.8148975372314453, 0.22909511625766754, 1.6546751260757446, 1.3385967016220093, 1.1814954280853271, 0.1928681582212448, 0.8200143575668335, -2.710082530975342, 1.5861685276031494, 1.043317437171936, -0.3336021900177002, -1.3062299489974976, -0.7954666614532471, 0.5816541314125061, -0.8042110800743103, 1.3942914009094238, 0.2546831965446472, -0.8181511759757996, -1.01967191696167, 1.3382763862609863, -2.2472946643829346, 0.3104729652404785, 0.3653312921524048, -0.7607031464576721, 0.37070760130882263, -0.12163592129945755, -0.31426072120666504, 0.24176865816116333, 0.5376903414726257, -0.3851039707660675, -0.9021151661872864, -1.1817387342453003, -1.9783633947372437, -0.9450174570083618, -1.3354729413986206, 0.777123212814331, 0.18452861905097961, -0.24561166763305664, 0.4600417912006378, -0.021558811888098717, 0.4085921347141266, -0.32238373160362244, 0.5417093634605408, 0.6948975324630737, -1.498341679573059, -0.30852264165878296, 1.79087495803833, 1.2642780542373657, 1.8235933780670166, 2.2719149589538574, 0.6677554249763489, -0.3875570595264435, -0.4874614179134369, 0.9472899436950684, -0.026064680889248848, 0.6111111640930176, -0.7660053968429565, 0.9426534175872803, -1.7644319534301758, 0.19947923719882965, -0.03254830464720726, -0.31383687257766724, -0.46773067116737366, 1.2006337642669678, 0.6035221219062805, -0.1341736614704132, -2.4612460136413574, 0.05466241016983986, 1.7390037775039673, 0.23147344589233398, -1.5662310123443604, 0.3121384382247925, -1.2055072784423828, -0.7611093521118164, -0.9806892275810242, 0.7741273045539856, 0.40294137597084045, 0.8258913159370422, 0.5084062218666077, 0.557992696762085, -0.40111956000328064, -0.23328982293605804, -0.126112699508667, -1.139953374862671, 0.1359645277261734, 0.40938055515289307, 0.4388086199760437, 0.1919328272342682, -0.7896623611450195, 0.5689675807952881, -0.2976764738559723, -0.8306729793548584, 0.9125322699546814, -1.6828769445419312, 2.0564754009246826, 0.7363130450248718, 0.3675095736980438, 0.8966436386108398, -1.0652958154678345, -0.0924830436706543, 1.6138904094696045, -0.22155413031578064, 1.4140459299087524, 0.736535906791687, 0.5587695240974426, 1.0497578382492065, 0.7713073492050171, -0.3413369059562683, 0.058058712631464005, 0.06485405564308167, 1.6555249691009521, -0.7077533602714539, 0.28600114583969116, 0.14675021171569824, 0.15493939816951752, -0.8256862759590149, 0.2621100842952728, 0.6749285459518433, 1.2890172004699707, -0.4413466453552246, -0.206624373793602, 0.2903580367565155, -0.06116178631782532, -0.1134023666381836, 0.5563117265701294, -0.14881862699985504, 1.100142240524292, 0.5442603230476379, 0.2247159481048584, -1.160904049873352, 1.5522340536117554, 0.1748843640089035, -1.4633100032806396, 0.09013382345438004, 1.188181757926941, 1.089444875717163, 0.610287606716156, -1.472760558128357, 1.4374970197677612, -0.767989993095398, -0.5683841109275818, -0.9528141617774963, 0.9567652344703674, -2.1100926399230957, 0.2851699888706207, 0.38255417346954346, 2.219198226928711, 0.29405710101127625, 0.5019105672836304, 0.18011140823364258, -1.5074094533920288, 0.20782437920570374, -0.15818709135055542, 1.1717851161956787, -1.1957595348358154, 3.5223419666290283, -1.545610785484314, 1.2376291751861572, 0.27690863609313965, -1.357938289642334, 0.250024676322937, 0.26878657937049866, 0.3527890741825104, 0.16341336071491241, 0.36965832114219666, -0.8457869291305542, -2.199845314025879, 1.4061174392700195, 0.8772826194763184, -0.8907009959220886, -0.6490747928619385, 1.0861965417861938, 1.5587742328643799, 1.854286789894104, 1.6917433738708496, -1.8873519897460938, 0.43438321352005005, -0.5678458213806152, 0.36628443002700806, -0.18030743300914764, -0.15299107134342194, -0.5172438025474548, 0.8996573090553284, 1.2889461517333984, 0.8513956665992737, 0.7653282880783081, -0.29510027170181274, 0.44894886016845703, 0.12497424334287643, 0.027665140107274055, -0.33916759490966797, -0.5342745780944824, 0.7508541941642761, 0.7931572198867798, -0.3689032196998596, 0.8451396226882935, -0.35937926173210144, 1.629809856414795, -2.127459764480591, -0.9885806441307068, 0.16145317256450653, -0.8481371402740479, -0.4693153202533722, 0.2304421216249466, 1.1438361406326294, 0.9130726456642151, 0.007919196039438248, 0.6753389835357666, 0.343667209148407, 1.306604027748108, 1.4667837619781494, -0.2634269595146179, -1.0592517852783203, 0.22251862287521362, 0.11202309280633926, -0.18622834980487823, -0.03535960987210274, 0.46601006388664246, 0.09832198172807693, 0.26953038573265076, 0.332395076751709, 0.8979688882827759, -1.9222726821899414, -0.3103831708431244, -0.34937164187431335, -0.9691314697265625, -1.5497642755508423, -0.09749724715948105, -1.3629175424575806, -0.9949027895927429, 0.16119562089443207, -0.7247602343559265, -0.549933135509491, 1.0005701780319214, 1.6612335443496704, -0.9621995091438293, -1.3126307725906372, -0.47873809933662415, -0.6093453764915466, -0.11419704556465149, 1.448008418083191, 1.346096396446228, 0.823605477809906, 0.3513143062591553, -0.5295137763023376, -0.2131194919347763, -0.717680811882019, -0.1256328821182251, -1.1556087732315063, -0.7552228569984436, -1.9129328727722168, 0.01840379647910595, 1.4831523895263672, 0.013753736391663551, -1.8517041206359863, 1.7881733179092407, 1.654181718826294, -1.1303563117980957, -1.013437271118164, 1.1347311735153198, 1.223576307296753, 0.3493969142436981, 0.5245543718338013, -0.6370130777359009, -0.027821334078907967, -0.8744156956672668, 0.8118334412574768, -0.06104103848338127, -0.5072388052940369, 0.5002886056900024, 1.701460838317871, -0.5595461130142212, 0.9411829710006714, -0.30373117327690125, -0.14887858927249908, 0.07306690514087677, -0.5651913285255432, 0.25506407022476196, 0.9058681130409241, 2.351231813430786, -1.0835866928100586, 1.1637545824050903, 1.377866506576538, 0.4496541917324066, 0.26936858892440796, -0.31468358635902405, -0.09956472367048264, 0.15877656638622284, 0.3547155261039734, 0.06083160266280174, 0.4562699496746063, -0.8993512988090515, 1.922747254371643, -1.189056396484375, 1.1696995496749878, 0.19182062149047852, -0.9199817776679993, -0.18638871610164642, 1.1636083126068115, -0.45498397946357727, -1.0202605724334717, 0.9038807153701782, -0.5193167328834534, -1.6906880140304565, -1.149445652961731, 1.2918660640716553, 0.08123341202735901, 0.7874011397361755, 1.2422305345535278, -0.2795560657978058, 0.7450344562530518, -0.24650157988071442, -0.19250835478305817, 1.01984441280365, 0.3618422746658325, -0.9405595064163208, 0.6456051468849182, -1.188852071762085, 0.6357314586639404, 0.8659769296646118, -0.6910445094108582, -0.7215179800987244, -1.142287254333496, 1.293771743774414, -0.16678377985954285, -1.113627552986145, -1.3360402584075928, 0.7692034840583801, -1.4912705421447754, -0.1844651997089386, 0.6483106017112732, 0.20321524143218994, 0.7452574968338013, -2.22983717918396, -0.027781175449490547, -0.3619749844074249, 1.5625388622283936, 1.3682180643081665, 0.004457780160009861, 0.23706774413585663, -0.2119399756193161, 1.7529276609420776, -0.7298775911331177, 0.16427774727344513, -0.04988281801342964, -3.7506208419799805, 0.2959105372428894, 0.36217641830444336, 0.27421408891677856, 0.3674519956111908, 2.078200578689575, -0.5946452021598816, -0.03447824716567993, -1.1128928661346436, -0.9967344403266907, 0.17510777711868286, 0.18668422102928162, -1.3601018190383911, 0.20700779557228088, 0.9748560190200806, -0.5979034304618835, -0.42606276273727417, -0.7206680178642273, 0.047833334654569626, 0.980266273021698, -1.2543766498565674, 1.0922257900238037, 0.21268415451049805, 0.33162713050842285, 0.9843103289604187, 1.0009516477584839, -0.44717657566070557, -0.45775339007377625, -0.0757935494184494, -1.907236099243164, 0.6597397923469543, 1.7818620204925537, -2.506436824798584, 0.9201370477676392, 0.02449958398938179, -0.12201552838087082, 1.5953638553619385, 0.5795025825500488, -1.7645541429519653, 1.135164499282837, 0.13249194622039795, 0.9199600219726562, -0.0046777441166341305, 0.8528397679328918, -1.2351102828979492, 0.14924539625644684, 0.946566104888916, 0.6236945986747742, 1.2107864618301392, 0.7005419135093689, 0.15448865294456482, -0.5255701541900635, -0.137550950050354, 1.8619225025177002, 0.8261342644691467, 0.4993895888328552, 0.7730386257171631, -0.9295066595077515, -0.45232149958610535, -0.058495666831731796, 0.802886962890625, -1.3242741823196411, 0.5721263885498047, 0.866409420967102, 0.5901235938072205, 0.22855226695537567, 0.3087851405143738, 1.1746524572372437, -0.6394059062004089, -0.46931520104408264, 0.1648324579000473, -0.97818523645401, -0.5997083783149719, 2.22609543800354, -0.9258338212966919, 0.004150460474193096, 0.7207010984420776, 0.7880347371101379, -0.16497553884983063, 1.123958706855774, -0.816775381565094, -1.242243766784668, 1.181959629058838, -0.1087624654173851, 1.0337095260620117, -1.1395022869110107, 1.3997247219085693, -0.007198079489171505, 0.27042892575263977, 1.3320786952972412, 0.8978898525238037, 1.053056001663208, 0.60016268491745, -0.640875518321991, -0.05400695651769638, -1.6542906761169434, -1.7566593885421753, -0.059678010642528534, -0.5268423557281494, -1.3558789491653442, 0.5604153871536255, 0.8514466881752014, 0.09302501380443573, -1.2290680408477783, -0.6812005043029785, 0.72031170129776, 1.9116172790527344, -2.3958725929260254, 1.1896039247512817, -0.8697405457496643, 0.6791145205497742, 1.26483154296875, 0.3513137400150299, 0.36027246713638306, 0.8791258931159973, -1.1684389114379883, 1.029889702796936, 1.1664830446243286, -1.7387409210205078, 0.43594038486480713, 0.6392940878868103, -0.968968391418457, -1.0543632507324219, 1.344650149345398, -1.6245099306106567, -0.06169482693076134, 0.3526034951210022, -0.9270155429840088, -1.2088634967803955, -0.4440981149673462, 2.541553258895874, 0.758808434009552, -0.586051881313324, -1.4106934070587158, 0.6347824931144714, 0.16821445524692535, 0.36865630745887756, -1.2280017137527466, -1.8322265148162842, 1.3581546545028687, -0.39008283615112305, -1.0898021459579468, 1.2694185972213745, -0.8901930451393127, -0.6715736985206604, 0.09530878067016602, 1.548403263092041, 0.45902904868125916, -0.5264846086502075, -0.7320088744163513, -0.06813362240791321, 1.8728362321853638, 0.3588956892490387, 0.3545432984828949, -1.1223407983779907, 0.2767830789089203, -0.5740317702293396, -0.17302486300468445, -1.1428172588348389, -0.910235583782196, 1.11378014087677, -0.2656257450580597, 0.031061440706253052, 0.26420366764068604, -0.4560297131538391, 0.6245933175086975, -1.588276982307434, 2.1252334117889404, 1.7896140813827515, -2.5861175060272217, 1.046094298362732, 2.0138492584228516, -0.8945798873901367, -1.3386666774749756, 1.011792778968811, 1.0332366228103638, 1.1896251440048218, -1.3177303075790405, -0.7016255259513855, 1.0779043436050415, -1.1871440410614014, 0.48751214146614075, 0.33847668766975403, -0.13647828996181488, -1.333324909210205, -1.5392512083053589, 1.2414112091064453, 1.4824252128601074, 0.41537314653396606, 0.8363437056541443, -0.6029302477836609, -1.1707751750946045, 0.1941300928592682, -0.7086465358734131, -0.0501682311296463, 0.10820319503545761, -1.0629167556762695, -0.6272140741348267, 1.5262871980667114, 0.07527739554643631, 0.6341344714164734, 1.0827782154083252, 0.4878823459148407, -1.4891949892044067, -0.8611103892326355, 1.0973771810531616, 0.3748535215854645, -1.4822018146514893, 0.5273429751396179, -0.8525689840316772, 1.1816221475601196, -1.039536952972412, -0.686564564704895, -0.5175228714942932, 0.0356634147465229, 1.5208871364593506, 0.5739001035690308, -0.2701379358768463, -1.9384765625, -0.8169911503791809, 0.521832287311554, 0.26804834604263306, 0.011399329639971256, -0.35320883989334106, 0.9738795161247253, 1.3083078861236572, -1.395087480545044, -1.0604625940322876, 1.6830836534500122, 0.45558568835258484, -0.38829004764556885, -0.7381018996238708, 0.3499186038970947, -1.085351824760437, 1.8737024068832397, 0.24045740067958832, 0.9813409447669983, 0.14911788702011108, -1.6195578575134277, -0.7165766358375549, -0.38796910643577576, -0.17233771085739136, -0.8287169337272644, 0.3007245361804962, 1.615212082862854, -1.600146770477295, -0.9424689412117004, -0.14861847460269928, 2.1645429134368896, 0.058810796588659286, -0.3826451003551483, 0.9636534452438354, -0.35853296518325806, -1.5758169889450073, -0.4753429591655731, -1.3104298114776611, 0.2279909998178482, 0.07528439164161682, -0.5125166177749634, -0.019014442339539528, 0.33344021439552307, -0.7922181487083435, 1.1428723335266113, -0.3288082182407379, -0.7065620422363281, 0.696377158164978, 0.2648658752441406, -0.27900567650794983, 0.7091031670570374, -1.0842127799987793, -0.7897838354110718, -0.37153685092926025, -0.00397298252210021, 0.9904499053955078, -1.6093477010726929, 0.7083475589752197, 1.7531615495681763, -1.1733983755111694, -0.49237826466560364, 0.9152893424034119, 0.8322965502738953, 0.5430553555488586, -1.854224443435669, 2.362039804458618, 2.3297507762908936, 0.3328368663787842, 0.5359857082366943, 0.26327431201934814, -0.2798251211643219, 0.47644665837287903, 0.07533835619688034, -0.25037989020347595, 0.305385947227478, -0.2624722421169281, 1.8787857294082642, 0.7901659607887268, -0.24017134308815002, -0.3646937608718872, 0.6994169354438782, -0.4557250440120697, 1.2917160987854004, -0.29495474696159363, -1.380614995956421, -0.5518296957015991, 1.0465998649597168, -0.5327154994010925, -1.2311166524887085, 1.228575587272644, -1.3051815032958984, -0.09009762108325958, 1.2530747652053833, 0.5117831826210022, -0.40133407711982727, -0.03227223455905914, 0.9990277290344238, -0.6925159692764282, -0.2937237024307251, 1.0013386011123657, -0.6230538487434387, 0.865158200263977, -0.44972386956214905, 1.510111689567566, -1.5915497541427612, -0.5102606415748596, 0.06505784392356873, -0.2896623909473419, -0.5294049382209778, -0.248099684715271, 2.7121243476867676, -0.9414410591125488, 1.2432000637054443, 1.3443119525909424, 1.840871810913086, 0.5467589497566223, 0.8157642483711243, -1.496201515197754, -1.41680109500885, -0.5593195557594299, 0.6698703765869141, -0.6025020480155945, -0.5672282576560974, -0.527982771396637, -1.3579427003860474, 0.28647151589393616, 0.7845517992973328, -2.2446539402008057, 0.0019228683086112142, 0.2909620404243469, -0.39311540126800537, 0.7135083675384521, 0.5395460724830627, -0.6583045721054077, -0.8201888203620911, 1.0045474767684937, -0.01322871632874012, 0.3324161171913147, -0.5905666351318359, 1.4226583242416382, -0.36672738194465637, 0.4691030979156494, -1.0565763711929321, 0.721368670463562, 1.277414083480835, -1.6586406230926514, 0.02132236585021019, -1.0359466075897217, 1.2838846445083618, 0.4729018211364746, 0.4980847239494324, -1.8658115863800049, -0.299562007188797, -0.8560904264450073, -1.1872830390930176, 0.3460516035556793, 0.028700297698378563, 1.1458395719528198, 1.812920331954956, 0.14059890806674957, 0.7543483376502991, -0.4450245797634125, 0.46792498230934143, 2.500872850418091, 0.6212683916091919, 0.01617409475147724, 0.6726474165916443, 1.1818729639053345, 1.2169420719146729, 0.1304849088191986, -0.9496193528175354, -0.8472057580947876, -1.970749020576477, 0.8569473028182983, -0.5312891006469727, -1.54385244846344, 0.5329227447509766, -2.032548189163208, -1.1223971843719482, 1.136383056640625, -0.6078003644943237, -1.3121657371520996, 0.15078407526016235, 0.6535086035728455, -0.5895391702651978, 0.3424852192401886, -0.24955469369888306, 1.880785346031189, -0.05848516896367073, -1.9582358598709106, -0.38272783160209656, 2.748910903930664, 1.0918983221054077, -0.5765140652656555, -1.176061749458313, -0.7933520674705505, -0.44209396839141846, 0.051360633224248886, 0.7501412630081177, -0.57921302318573, -1.9279745817184448, -1.2490276098251343, -1.2993035316467285, 0.05311468988656998, -0.6989889144897461, -1.2539646625518799, 0.45385223627090454, -1.450745940208435, 1.853012204170227, 0.19021070003509521, -0.40112531185150146, -2.219550132751465, -0.20045672357082367, 0.9044784903526306, -0.7414699792861938, 1.2115908861160278, 0.9568835496902466, 1.2944047451019287, 0.5773850083351135, 1.0958867073059082, 1.2887604236602783, -1.6756491661071777, -0.3646351993083954, -0.2811621427536011, -0.7819153666496277, 0.44285473227500916, -0.36192700266838074, -1.3518619537353516, -1.4251055717468262, -0.3038332462310791, -0.7589373588562012, -0.24016998708248138, -0.3224993944168091, -0.16023775935173035, 0.8997126221656799, 0.5828240513801575, -0.5202568769454956, -0.7016947269439697, -0.8290373682975769, 0.12400399893522263, -0.7792888283729553, 0.6046850681304932, -1.0869219303131104, 0.05210969224572182, -0.31976303458213806, -1.4465088844299316, -1.0663424730300903, 0.9083476066589355, -0.5185222625732422, -1.4973788261413574, 1.711029052734375, 0.18089914321899414, 0.5072816610336304, 0.7342672944068909, -0.18414491415023804, -0.05292551591992378, -1.2975435256958008, 0.9240834712982178, -0.00355786201544106, -1.467539668083191, 1.9299445152282715, -0.4241706132888794, -0.366672158241272, 0.33810749650001526, 0.10836589336395264, 0.25408732891082764, 0.1524258255958557, -1.6202116012573242, 0.6344475746154785, -0.687103807926178, -0.4331761598587036, -0.5449445843696594, 0.24101395905017853, -0.7008891701698303, -0.0803985446691513, 1.0933382511138916, 1.3907814025878906, 1.7096221446990967, -0.8807893395423889, -0.10928677767515182, -1.7069430351257324, 0.688371479511261, 0.21679437160491943, -0.3505138158798218, 0.4569408595561981, 0.6910094618797302, -0.427318811416626, 0.008868083357810974, -0.5196818709373474, -0.3436630368232727, 0.2955361008644104, -0.5807260870933533, 0.034378618001937866, 2.9168851375579834, 2.5650904178619385, -0.26171383261680603, -1.9141285419464111, 0.16500437259674072, 0.20385698974132538, -1.1680400371551514, -0.12825562059879303, 0.06942863762378693, -0.906981885433197, -0.1454382836818695, 0.9476553201675415, 1.1397868394851685, -0.12389814108610153, -0.7062733173370361, 0.8911926746368408, 0.4153854250907898, 0.716284990310669, 1.6341320276260376, -0.5044506788253784, -1.130629301071167, 1.1088868379592896, 1.120465874671936, 1.4242022037506104, 1.6152000427246094, -0.7413103580474854, -0.3800312280654907, -0.2176564335823059, 0.5042084455490112, 1.3665778636932373, -1.0445301532745361, 0.36641725897789, 0.3159462511539459, -1.2816706895828247, 0.22636207938194275, 0.3072986304759979, 0.1358724981546402, -2.0789670944213867, -0.059697654098272324, -0.7967281937599182, 0.19279685616493225, 0.07072044909000397, 0.034333862364292145, 0.6579593420028687, -0.46868520975112915, -1.0376054048538208, -1.8157072067260742, -0.2445385754108429, -0.4797748327255249, -1.1703680753707886, -0.426611989736557, -0.3157043159008026, -0.3652309477329254, 1.3906551599502563, -0.03962983936071396, 0.025509128347039223, -1.171739935874939, -0.9022650122642517, -0.6643670201301575, -0.5053341388702393, 1.763360619544983, -0.7916986346244812, -1.8122919797897339, 1.8829855918884277, 0.8848848342895508, 0.7901281118392944, -0.24683231115341187, -0.010782393626868725, -0.7725029587745667, -1.153003215789795, 1.5559570789337158, -0.5268287658691406, 0.3000125586986542, 0.0383259579539299, -1.362424612045288, 0.4081539511680603, 2.388770341873169, 1.6740370988845825, -1.0860711336135864, 1.6442500352859497, 0.23218950629234314, -0.8960049748420715, 0.6030701994895935, 0.3801117539405823, 2.037003993988037, 1.4253973960876465, -0.6558573246002197, 0.3958693742752075, -0.9839866757392883, -0.9316397905349731, 0.1352163404226303, -1.545886516571045, -0.052927590906620026, 0.5928663611412048, 0.47831740975379944, -0.6395667195320129, 0.20449993014335632, 2.8403992652893066, -1.576071858406067, -0.6032772660255432, -2.2308318614959717, -0.8995164036750793, 0.23433436453342438, 0.22850994765758514, 0.20837262272834778, -0.6045209765434265, -0.029628144577145576, -0.7172871828079224, 0.32274696230888367, 0.1326724886894226, 0.6220396757125854, 1.4190551042556763, 0.053669724613428116, 0.5582161545753479, 0.4283113181591034, -0.0261390283703804, 0.45880356431007385, -0.38506823778152466, -0.36017313599586487, 0.013606863096356392, 2.2421481609344482, -0.8443921804428101, 1.5236713886260986, 2.125566244125366, -1.5207099914550781, 0.3567304015159607, -1.3471585512161255, 0.586540162563324, 0.4741598665714264, 1.5138434171676636, 0.17619919776916504, -1.132346510887146, -0.218723863363266, 0.039026692509651184, 0.8103602528572083, -0.794120728969574, -1.4059337377548218, 1.4264960289001465, -0.3020242154598236, -0.14126740396022797, 1.4940623044967651, -0.7384267449378967, -0.14464445412158966, -0.5141178369522095, 1.1796026229858398, -0.49754464626312256, -0.05408093333244324, -0.28847044706344604, 1.0377827882766724, 0.4315154254436493, 0.6808634400367737, -0.42709288001060486, -1.4016244411468506, -0.7522410154342651, -0.13659216463565826, -0.43903011083602905, 0.8369160294532776, 1.0189989805221558, 0.6026085615158081, -1.507462501525879, 1.4646756649017334, 0.458224356174469, 1.7254469394683838, 0.5970233082771301, 0.1178373321890831, 1.601632833480835, -0.23227234184741974, 0.9153119921684265, -0.2653447985649109, -0.5014353394508362, -1.1504210233688354, 0.7461785078048706, -0.4393698573112488, 1.4030160903930664, 0.19247126579284668, -1.6327581405639648, 0.9244004487991333, -0.7165163159370422, 0.3563188314437866, 1.5078250169754028, 0.5897659659385681, -1.681807518005371, -1.2795830965042114, 0.6365281343460083, 1.2936245203018188, 1.2494020462036133, -2.5126991271972656, 1.0724745988845825, -0.3969078063964844, -1.5321121215820312, -1.5655083656311035, -0.6137297749519348, 0.1976289600133896, 0.5714439153671265, 0.5661506652832031, 1.0548186302185059, -1.4054101705551147, 1.1891053915023804, -1.1135337352752686, 0.09993503987789154, 0.24711738526821136, 0.22561192512512207, 1.2878000736236572, 0.9123519062995911, -0.9986217021942139, -0.002067301422357559, 1.3219648599624634, 2.210125207901001, -0.28530818223953247, 0.7709079384803772, -1.8095961809158325, -0.6983821988105774, 1.1172056198120117, -0.44712644815444946, 0.6900849938392639, -1.0630520582199097, -0.30298060178756714, -2.355123519897461, -0.03518610820174217, 2.249786376953125, 0.06758091598749161, -0.4860789179801941, 0.68093341588974, 0.1433933526277542, 0.9846422672271729, -3.031304359436035, -0.43860459327697754, -0.321576863527298, -0.07782560586929321, -0.32974016666412354, 0.9293506741523743, 0.117493636906147, -0.11799894273281097, -0.957339346408844, 0.6460901498794556, -2.016706943511963, -0.13520902395248413, -0.03260399028658867, -0.4697757959365845, 1.5803377628326416, -1.2694849967956543, -1.5823743343353271, 1.2805060148239136, 0.8937628865242004, 0.3560413718223572, 0.9737850427627563, 0.4390398859977722, 0.8453173041343689, 0.019134951755404472, -0.04375327005982399, -0.0873207300901413, 1.307916283607483, -0.37419140338897705, -0.5922031998634338, -0.07906679064035416, -0.4477745592594147, 0.7303635478019714, -0.22279319167137146, -1.6506128311157227, 0.39061859250068665, -1.4481269121170044, -1.2119344472885132, 1.242801547050476, -0.0384872741997242, 0.10579860210418701, 0.0659896582365036, 0.08246006816625595, 0.6057083606719971, -0.4990249574184418, -0.5311809778213501, -1.124977946281433, -1.5200629234313965, 0.38804176449775696, 1.5471042394638062, 0.20021897554397583, 1.7924343347549438, -1.0412966012954712, 0.5196822881698608, -0.9699812531471252, -1.1588411331176758, 0.8682909607887268, -0.8082653880119324, -0.49618977308273315, 0.3462893068790436, 0.5641733407974243, 0.13734790682792664, 0.8081430792808533, -0.7400351166725159, 0.43179404735565186, 0.8558101058006287, -2.1061434745788574, 0.3720257580280304, 0.6942050457000732, 1.217272400856018, -0.44380465149879456, -1.945073127746582, 0.1467633992433548, 0.9153527021408081, -0.5414685606956482, 2.4685165882110596, 2.5482077598571777, 0.056433141231536865, 1.0908094644546509, 1.7721686363220215, -1.9400256872177124, -0.176893949508667, 0.4544902741909027, -0.1969357430934906, -1.846595048904419, 0.82673579454422, 1.1181081533432007, 1.2459195852279663, 0.1317680925130844, -0.38895300030708313, 1.1015478372573853, -1.3102830648422241, -0.867365300655365, 1.5319187641143799, 0.842811644077301, -0.2829672694206238, -0.6334571242332458, 0.5274494290351868, 2.214118719100952, -0.10522783547639847, -0.9127485752105713, -1.4699369668960571, -1.543558120727539, 0.1433086395263672, 0.1410994529724121, -0.34006139636039734, 0.17838560044765472, -0.023518605157732964, -0.14411836862564087, -0.10160962492227554, -1.1364307403564453, 0.997591495513916, -1.6200547218322754, 0.2067854255437851, 0.20707854628562927, 0.3694901764392853, -0.7002573609352112, -0.17183250188827515, 1.498839020729065, -0.3497072160243988, -0.47119811177253723, 1.3193788528442383, -0.9582213759422302, -0.0030946340411901474, 1.1087638139724731, -1.4743402004241943, -1.401694655418396, 1.2727668285369873, -0.2704099714756012, 1.3641973733901978, -0.8768600225448608, -0.5854210257530212, 1.0505212545394897, -0.7285435199737549, -0.3877694308757782, 1.13167405128479, 0.41141581535339355, 1.1157523393630981, 1.9293193817138672, -0.48863404989242554, 0.0012855792883783579, 1.5875920057296753, -0.1354319304227829, 2.283607006072998, 0.7791032791137695, -1.6148488521575928, 0.3633112609386444, 0.23456613719463348, 0.974425733089447, 0.45848849415779114, 1.0244286060333252, -0.10173455625772476, 1.1445395946502686, -0.3393637239933014, -1.027182698249817, -1.2647802829742432, 0.02155320532619953, -0.3756297528743744, 0.17168158292770386, -0.18216902017593384, -1.2669479846954346, 0.014777442440390587, 0.9106634259223938, -0.16790418326854706, 0.1281014382839203, 2.3079171180725098, -0.6076154112815857, -1.7721959352493286, -0.0033172171097248793, 0.7979966402053833, -1.2721548080444336, -0.13503769040107727, 1.876559853553772, -0.12055559456348419, 0.7738337516784668, 0.04652947559952736, 0.7021233439445496, -1.1813851594924927, -2.309872627258301, 0.22626899182796478, 2.111637592315674, 1.6835159063339233, 0.5042433142662048, 1.3220865726470947, 0.06857805699110031, -0.8369140028953552, -0.5228655338287354, -1.4211570024490356, -0.34628912806510925, -0.17820966243743896, 0.9862743616104126, 0.8298584818840027, -1.7867205142974854, 1.0515440702438354, 1.146396517753601, 0.24211359024047852, 0.19475647807121277, -1.8675254583358765, -1.0371309518814087, -0.228899285197258, 0.166327103972435, 1.541662573814392, 1.2359163761138916, 1.1359970569610596, 0.425878643989563, -0.5340747833251953, -0.7959315776824951, -0.8823422789573669, -0.5331218242645264, 1.4784208536148071, -0.051462966948747635, -0.02188475988805294, -1.8164751529693604, 0.4228239357471466, 0.42399466037750244, -0.3317440450191498, 0.42689457535743713, -0.07893316447734833, -0.4831063449382782, 1.0385922193527222, 0.4388943314552307, -0.010477310977876186, 0.6701818704605103, -0.2986793518066406, 0.9638031125068665, -0.011960663832724094, 0.26429757475852966, 2.0108642578125, 1.3065226078033447, 1.6770931482315063, 1.5044816732406616, 0.4568330645561218, -1.0427268743515015, 1.661896824836731, -2.1495068073272705, -0.6159226298332214, -0.5696969628334045, 0.7241474986076355, 1.1075457334518433, 0.14653338491916656, 0.6417097449302673, -0.13952945172786713, -1.1916210651397705, 0.15692126750946045, -0.9133349061012268, -0.3682903051376343, -1.2566323280334473, -0.23542840778827667, -0.5252189040184021, 0.8355576992034912, 0.711596667766571, 1.5306719541549683, -0.04894590005278587, -0.27540239691734314, 1.3650728464126587, 1.591212272644043, -1.1809415817260742, -0.9229867458343506, 0.8831787109375, -0.6565451622009277, 0.2701084017753601, 0.3230404853820801, 1.1625475883483887, -0.2981424629688263, 0.6184073686599731, -0.20113763213157654, -0.881788969039917, 0.5331671833992004, 0.4106343388557434, 0.35521337389945984, 1.462038516998291, 0.041733406484127045, -1.4869624376296997, 0.023501388728618622, -0.5699203014373779, -0.7591404318809509, 0.9760475158691406, -0.4920442998409271, -2.2193195819854736, -0.4005435109138489, -0.058453600853681564, 1.4851053953170776, -1.3520034551620483, -1.7772465944290161, 0.6142017841339111, -0.37822675704956055, 0.807327151298523, -0.3503934442996979, -1.1988062858581543, -1.3775583505630493, -0.501115620136261, 0.5633214116096497, -1.210681438446045, 0.12358549237251282, 0.7495800852775574, -0.9154694080352783, -0.10830432176589966, -1.2867255210876465, 1.8765050172805786, -0.3852287530899048, -0.5488672256469727, 0.9106056094169617, -0.35632988810539246, 1.9573131799697876, -0.4492068290710449, -0.23337508738040924, 1.4352116584777832, 0.3301924467086792, 0.7918685078620911, 1.2238268852233887, -1.6734596490859985, 1.0385724306106567, -0.7598176598548889, 0.3231239914894104, 0.6635652184486389, 1.2081401348114014, 0.8319656252861023, -0.4891505837440491, -0.09946563094854355, -0.7116336822509766, -0.317050576210022, 0.8678260445594788, 0.6922799348831177, -0.15549437701702118, -0.30234208703041077, -0.2552598714828491, 0.049628064036369324, -0.5624845623970032, 0.9122341871261597, -0.2986432611942291, 0.21924325823783875], [-0.30626943707466125, 0.997707188129425, 0.8974095582962036, 0.9386290311813354, 0.11198961734771729, -0.9746180772781372, 1.3662930727005005, -0.5299767255783081, 0.1555764228105545, -0.21674053370952606, -0.3897847831249237, 0.4863434433937073, 1.5968364477157593, 0.355329692363739, -1.1829171180725098, -0.31186357140541077, -0.6467381715774536, 0.695815920829773, 0.18757575750350952, -0.09163361042737961, -0.03859017789363861, 0.9643301367759705, 1.4111160039901733, -0.5052359700202942, 1.2019023895263672, 0.2856674790382385, 0.1677444726228714, 0.5057482719421387, 0.2186630517244339, -0.5067522525787354, 0.1905643343925476, -0.7574199438095093, -1.4356689453125, -0.6433265209197998, -0.15104736387729645, -0.6837127208709717, 2.2443313598632812, -0.07953702658414841, 0.3218388855457306, -2.102324962615967, -1.0432515144348145, -0.4550255537033081, 2.4686131477355957, 0.4497739374637604, 0.009969891048967838, -0.2320672571659088, -1.0193672180175781, 0.2604089379310608, 0.8362262845039368, -0.7636758685112, 0.6558315753936768, 0.17712298035621643, -0.9311532378196716, -0.6403496265411377, -0.31546705961227417, 0.5614141821861267, 1.028555154800415, -2.2905938625335693, -1.2193862199783325, 0.00844729132950306, -0.1301683932542801, 0.26508209109306335, -0.610893964767456, 0.6194338202476501, -0.37911057472229004, 0.7149343490600586, -0.8567208647727966, 2.2684648036956787, 1.6129781007766724, -1.5974386930465698, 1.2913331985473633, 0.7207849025726318, 0.5943710207939148, -0.23780381679534912, 0.945700466632843, -2.323439121246338, 2.1624181270599365, 0.21083232760429382, -1.0780400037765503, -0.0010399656603112817, -0.7516841292381287, 1.2908744812011719, 0.10858238488435745, -0.9137144088745117, 0.31297245621681213, -1.0271974802017212, -0.7308693528175354, -0.13109968602657318, 0.3271806240081787, 0.638152539730072, 0.6241794228553772, -1.7784852981567383, 0.11339972168207169, -0.39322444796562195, -0.808447003364563, -0.3636566698551178, 1.0010018348693848, 1.176578164100647, 0.029404448345303535, 0.3348907232284546, 0.3251042068004608, 0.37620115280151367, -0.12972640991210938, 0.21468575298786163, 0.7161930203437805, 0.6299254894256592, -0.9433961510658264, -1.3173733949661255, -1.196667194366455, 1.6173129081726074, -0.4686083495616913, 0.366809219121933, 1.694450855255127, -1.0453786849975586, 0.04516409710049629, 0.8159224390983582, -0.724560558795929, -1.297470211982727, 0.09251090884208679, 0.5367145538330078, 0.13112406432628632, -0.48240017890930176, -0.9271553158760071, -1.0250645875930786, 1.8433351516723633, -0.08612947911024094, -2.2082581520080566, -0.33063724637031555, 1.86093008518219, 0.9539127945899963, -1.3795663118362427, 0.06965398043394089, -0.4509682357311249, -0.3583778142929077, 1.6227351427078247, 0.08321196585893631, 1.5152091979980469, 1.4309375286102295, 0.46675965189933777, -1.0050081014633179, 0.36450475454330444, 0.7573446035385132, -0.4800563156604767, 0.321723073720932, 2.179112195968628, 0.6615631580352783, -1.018504023551941, 0.1009894460439682, 0.07038616389036179, 1.4370635747909546, 0.7396204471588135, -0.15722060203552246, -2.0896997451782227, -1.1415716409683228, -0.8266239166259766, -0.37922245264053345, -1.05547034740448, -0.13431258499622345, 1.5851655006408691, -1.9076818227767944, -1.8656760454177856, 0.5369816422462463, 0.6616964340209961, -1.857608437538147, 0.4835250675678253, -0.7773807048797607, 1.1572355031967163, -1.0711857080459595, 0.33233213424682617, 1.4322510957717896, -0.7971236705780029, -0.4222855269908905, 0.4825785756111145, -2.3023359775543213, -0.7301530241966248, 0.25981801748275757, 1.7355506420135498, -0.40296512842178345, -0.4769611060619354, -1.968615174293518, -0.3208543062210083, 0.7332196235656738, 0.6205493807792664, -0.030728965997695923, -0.11004018783569336, -1.83894681930542, 0.8187444806098938, -0.3832290768623352, 0.200384721159935, 0.7490919232368469, -0.4625709652900696, -1.214423656463623, -0.8171597719192505, -0.034449588507413864, -0.9375916719436646, -0.1267978399991989, 0.21683426201343536, -0.5619096159934998, 0.5020999312400818, -0.0922698974609375, -0.321537047624588, 0.5622417330741882, 0.7297956943511963, -0.30788055062294006, -1.2393240928649902, -0.4323700964450836, -2.2101383209228516, -1.0160930156707764, -1.491713047027588, -1.3827366828918457, -1.1196966171264648, -0.2858535647392273, 0.7068396210670471, -0.040887657552957535, -1.5583552122116089, -0.4000307321548462, -1.7992408275604248, 0.6336855292320251, -1.1175458431243896, 0.41683316230773926, 0.7400457262992859, -0.855082631111145, -0.6397275924682617, -0.1122727021574974, -0.5085621476173401, -0.3051437735557556, 0.4982645511627197, -1.2133879661560059, 0.6076703071594238, 0.49245837330818176, 0.034216396510601044, 0.03403449431061745, -2.2479331493377686, -1.3765093088150024, -0.7888764142990112, 0.9894916415214539, -0.5143917202949524, -0.09844374656677246, 1.2890690565109253, 0.3145737946033478, -0.5889947414398193, -0.5744021534919739, 0.680340588092804, -0.3435092568397522, -0.4414826035499573, -0.42748475074768066, 0.004753139801323414, -0.022686056792736053, -1.4726619720458984, 0.4362213611602783, 0.932530403137207, 1.1855227947235107, 0.300610214471817, -0.914647102355957, -1.941945195198059, 1.1978245973587036, -0.044808488339185715, 0.22645807266235352, 0.5058602094650269, 0.7540509104728699, -0.504002571105957, 0.6402392387390137, -1.7195910215377808, 0.4610707759857178, -1.0543304681777954, -0.8277673721313477, -0.372721403837204, 0.7181453108787537, -1.461526870727539, -0.16385620832443237, 0.7511706352233887, -0.5628148317337036, 1.8987981081008911, 0.618594765663147, -0.5783621072769165, 1.35637629032135, -1.1695152521133423, -0.0676853358745575, -1.1300276517868042, 1.5158100128173828, 0.4746522307395935, 1.0574870109558105, -1.863480806350708, -0.9594829678535461, -0.7565041780471802, 0.08373194932937622, 0.14780399203300476, -0.5958805680274963, -1.408862829208374, 0.33789312839508057, 0.7041674256324768, -0.2128382921218872, -1.0364476442337036, 0.11610565334558487, -2.4464704990386963, 1.617438793182373, 0.5462415814399719, 1.2433691024780273, -0.02153848484158516, -1.6124283075332642, 0.2412683367729187, 1.1631412506103516, -0.4235774576663971, -0.2693803012371063, -0.7298232316970825, 0.41073286533355713, -0.2968437671661377, -0.829375147819519, 2.1329946517944336, -1.0351531505584717, 1.4394594430923462, 0.5089556574821472, 0.021014370024204254, 0.6177688241004944, -0.6810114979743958, 1.2186421155929565, -0.15856611728668213, 0.27681562304496765, -1.2165331840515137, -0.49031174182891846, 1.560874104499817, -1.5027695894241333, -1.769192099571228, 0.4616978168487549, 1.283200979232788, 0.7078384160995483, 1.232123851776123, 0.8633943796157837, 0.310134619474411, -0.365739107131958, 1.6187206506729126, -0.8081948161125183, -0.3306964635848999, 0.45019102096557617, -0.4205732047557831, -0.06973223388195038, 0.10996685177087784, 0.052415721118450165, -0.1679220199584961, 0.31312283873558044, 0.5209329128265381, -1.5543009042739868, -1.3396155834197998, 1.4249211549758911, -0.30134016275405884, -0.03903578966856003, -0.8914971947669983, -0.14276424050331116, 0.4322703778743744, 0.5474892854690552, 0.3164026737213135, -0.027071291580796242, -0.0335996188223362, -0.9336338639259338, -0.10781385749578476, 0.5383645296096802, -0.6296219825744629, 1.1220585107803345, -0.06352770328521729, 0.6422670483589172, 1.3106369972229004, 0.1532365381717682, -0.45298895239830017, 1.8044885396957397, -0.12258109450340271, 0.20790016651153564, -0.8418092727661133, -0.2233470231294632, -0.09344644844532013, 0.44587764143943787, -1.5015449523925781, -1.2084102630615234, -0.509518027305603, 1.3155264854431152, -0.9136859178543091, 0.4104974567890167, -1.2044216394424438, 0.1923898458480835, 1.2588565349578857, -0.564548134803772, -0.8299391865730286, -0.37950512766838074, 0.7508776187896729, 0.14898787438869476, -0.9045079350471497, -1.4786592721939087, 1.3365962505340576, -0.2369079738855362, -1.0025629997253418, -0.874919056892395, -0.1929587870836258, 0.7853439450263977, 0.14255733788013458, 1.4033535718917847, 2.64520263671875, 1.8921259641647339, -0.10830391943454742, -0.7098429203033447, 0.5267274975776672, -0.81859290599823, -1.1940635442733765, -0.49615541100502014, 0.3253461718559265, -0.34834712743759155, -0.9015970826148987, -1.099998950958252, -0.32318782806396484, -0.3744339644908905, -1.7261515855789185, 1.5677546262741089, 0.8365795612335205, 0.7423462271690369, 0.7334597110748291, -1.1434255838394165, 0.2548935115337372, -0.5073075890541077, 0.4088091254234314, 0.43008846044540405, 1.3383376598358154, -0.4929526448249817, 0.8243233561515808, 0.2233932912349701, 0.6887323260307312, 0.7456619739532471, 0.48274943232536316, -0.05633755400776863, -1.1096402406692505, -1.1135066747665405, 0.07500390708446503, -0.5839526057243347, -0.8324493169784546, 1.3905192613601685, -0.835528552532196, 0.04679442197084427, 0.7046358585357666, -0.853954017162323, -1.7121292352676392, -1.4498885869979858, -1.162671446800232, 1.2208107709884644, -0.2338605374097824, -0.9020105004310608, 1.4667593240737915, 0.036033473908901215, -2.098670244216919, 1.1359509229660034, -0.6627845168113708, 0.9308309555053711, 0.5973419547080994, -1.0197806358337402, -0.39481836557388306, 0.0003072099352721125, -0.44367894530296326, -1.1253618001937866, 0.7552692890167236, -0.22378911077976227, -0.2776073217391968, -1.0487406253814697, -0.19600361585617065, -1.1468557119369507, -0.8843593001365662, 1.0795098543167114, -0.09258557111024857, 0.019938930869102478, 1.3187731504440308, 0.6047413945198059, -1.2352186441421509, -0.5160769820213318, 0.9818949103355408, 0.26571646332740784, -0.7249745726585388, -0.9805247783660889, -1.4820778369903564, 0.4159451425075531, -0.09767021238803864, 0.24113184213638306, 0.7177504301071167, -0.4797273278236389, 2.2961137294769287, -1.3273472785949707, 1.1937583684921265, 0.6367756128311157, 1.4462758302688599, -0.13856667280197144, 0.6814826726913452, 0.3026355504989624, 0.831552267074585, -0.7641222476959229, 0.33261728286743164, -0.37237250804901123, 0.8697189092636108, -0.14413300156593323, 1.1469740867614746, 1.3639893531799316, 0.3269345164299011, 0.18855056166648865, 1.6710163354873657, -0.8994774222373962, -0.905825138092041, 0.009363323450088501, 0.7885271310806274, -1.0295628309249878, 1.1952614784240723, 1.4860519170761108, 2.430534839630127, -0.13085973262786865, 1.4372117519378662, -0.7534263730049133, 0.36963871121406555, 1.052947998046875, 0.07357075810432434, 0.27033549547195435, 0.7592614889144897, -0.023369064554572105, 0.5512493848800659, 0.1911381185054779, -0.1343739628791809, -0.9260401129722595, 0.0005736702005378902, 0.6000460982322693, 1.1720689535140991, 1.718658208847046, 1.9961284399032593, 0.7753622531890869, 0.02611817792057991, 1.9268120527267456, 0.9263246655464172, -0.3757520020008087, -0.3595791161060333, 1.0051119327545166, 1.454015851020813, 0.9363654255867004, -0.5758241415023804, -0.9409846663475037, 0.3966512978076935, -0.5229297876358032, 0.7991711497306824, 0.07796568423509598, 1.47702956199646, 0.15869836509227753, -0.021839531138539314, -0.9441710710525513, -0.601601243019104, -1.5075715780258179, 0.31508833169937134, 1.7795062065124512, 1.25269615650177, 0.2194564789533615, -0.2758393883705139, -1.5069084167480469, 1.4696577787399292, -0.965958297252655, 0.03452666476368904, 2.046579360961914, -0.8859478831291199, 1.6206750869750977, 0.1797153800725937, 1.2171553373336792, -1.9966671466827393, 0.2580353617668152, 1.21428382396698, -1.0515649318695068, 1.013088583946228, -1.4412258863449097, -0.06727030873298645, 0.05023939535021782, 1.1784820556640625, 1.4168412685394287, 0.38613829016685486, 0.6277468800544739, 0.29906097054481506, 0.15543285012245178, 0.5619442462921143, -0.6606974601745605, -1.799239993095398, 0.36795753240585327, -1.362463355064392, -1.5580179691314697, -0.47881051898002625, 0.06348787248134613, 2.1133904457092285, -0.8940222859382629, 0.4539645314216614, -0.8960977792739868, 0.3195718228816986, 0.24261631071567535, -1.755698800086975, -0.8474744558334351, 0.7070556879043579, -0.8848468661308289, -0.22098031640052795, -1.1514873504638672, -1.3894318342208862, -1.08989417552948, -0.5826398730278015, -2.3101141452789307, 0.3855474889278412, 0.12984336912631989, -1.3203165531158447, 0.4801926016807556, 0.5638256072998047, -1.1658694744110107, -1.1512993574142456, -1.0834460258483887, 0.00872834399342537, -0.12778832018375397, 0.943863570690155, -0.4611486792564392, -0.8112138509750366, -0.6418617367744446, -0.40449073910713196, 1.170397162437439, -0.07632729411125183, 1.0606508255004883, -0.6945741176605225, 1.0590217113494873, 0.591324508190155, -0.219233900308609, -0.18245261907577515, 0.14416702091693878, -0.09702136367559433, -1.0375878810882568, 1.711477518081665, 1.6602956056594849, -0.5174612998962402, 0.29743579030036926, 0.744584321975708, -0.2017713487148285, 0.15766632556915283, -1.1926031112670898, 0.1252928078174591, -1.3845462799072266, 1.4848666191101074, -1.1830335855484009, -0.8183285593986511, -1.1779859066009521, 0.8197864294052124, -1.4218398332595825, 1.5292198657989502, -0.7698395848274231, -0.9168894290924072, 0.3606574237346649, -2.456324577331543, 1.282549262046814, 1.441834807395935, -0.5148311257362366, -1.3401575088500977, -0.2589026689529419, 0.06063913181424141, 0.5215164422988892, -0.7821553349494934, -0.48949742317199707, 1.1724655628204346, 0.11475969851016998, -2.0501856803894043, -0.7267501950263977, -0.3719140887260437, -1.105354905128479, -1.529943585395813, 1.3399938344955444, -0.09670110791921616, 1.3542461395263672, -0.4327790439128876, -0.44812196493148804, -0.2407521903514862, 1.730691909790039, 1.0375264883041382, -1.034277081489563, 0.8028196692466736, -0.003043543780222535, -0.31325045228004456, 0.6889673471450806, -0.2214156687259674, 1.0109375715255737, 0.45192521810531616, -0.1430482715368271, -0.01349592860788107, -0.38637980818748474, 1.4024933576583862, -0.9007426500320435, -0.6550514101982117, -0.6044138073921204, -2.965787172317505, -0.1422530561685562, 0.265699565410614, -1.6061941385269165, -0.44195184111595154, -0.9339867830276489, -0.5778188109397888, -2.3523507118225098, -1.1486440896987915, -0.6811478734016418, 0.3665948212146759, 0.6361807584762573, 1.4052425622940063, -0.19619792699813843, 2.258293867111206, 0.17043156921863556, 0.5831817388534546, -0.6296765804290771, 0.9864898920059204, -1.5748978853225708, 0.9611122608184814, -0.6837133765220642, -1.1733245849609375, 0.8049459457397461, 1.1361504793167114, -0.319280743598938, -0.3225244879722595, -1.6874723434448242, 1.2646925449371338, -1.8151112794876099, -0.6470667123794556, -1.6219531297683716, -2.1145408153533936, -0.05233887955546379, 0.26628217101097107, 0.7044485211372375, 1.0895191431045532, -0.6004061698913574, 1.2480415105819702, -1.0497715473175049, -1.7506811618804932, 0.004761955700814724, -0.52754807472229, -0.0370059609413147, 0.5776951313018799, 0.21505418419837952, -1.3779592514038086, -0.8344358205795288, -0.21018795669078827, 0.025067675858736038, 1.1016614437103271, 0.7589171528816223, 0.9317578077316284, -0.5667921900749207, -1.4816161394119263, 0.7970032691955566, 0.3685082793235779, 0.2166348695755005, 0.09636793285608292, -1.2273119688034058, 1.313036322593689, 1.2729177474975586, -0.8687161207199097, -1.2817939519882202, -0.49794071912765503, 0.1486782729625702, -0.39309999346733093, 1.471665859222412, 1.4735329151153564, -1.5766088962554932, -0.97134929895401, -0.4722817838191986, 0.5908008217811584, -0.1590249091386795, -1.4065635204315186, 0.14172399044036865, 0.48890233039855957, -0.9773879647254944, -0.41393616795539856, -0.6506272554397583, -1.2706736326217651, -2.2074368000030518, -0.23506656289100647, 0.6064234972000122, -0.022790536284446716, -1.404276967048645, 0.5677065253257751, 0.26634687185287476, 0.6753032803535461, 1.4025503396987915, 0.04668877646327019, 1.2429088354110718, -0.6475653648376465, -1.5791372060775757, 0.5488048791885376, -0.283676415681839, -0.7757222056388855, 1.4099997282028198, -0.6147942543029785, 0.7921703457832336, 0.05453425273299217, -0.5000431537628174, -0.38127240538597107, 1.4435701370239258, -2.7501797676086426, 0.36432817578315735, 1.6699886322021484, -0.2813800573348999, -0.42207300662994385, -0.5676153302192688, 1.4117541313171387, 0.09621167927980423, -0.3662228286266327, -0.8504729270935059, 2.673142910003662, 1.3721551895141602, 0.3391847610473633, -0.989485502243042, 0.3488093912601471, -0.8035715222358704, -0.5695887207984924, 1.0476396083831787, -0.9071540236473083, -1.904584288597107, 0.8979916572570801, -0.4172058701515198, 0.32222622632980347, -2.3896872997283936, 1.6368266344070435, 0.26426056027412415, 1.0768393278121948, 0.6046473383903503, 0.363142728805542, 1.032608151435852, 0.5170049667358398, 0.2012859433889389, 1.228080153465271, 0.9233465194702148, 0.7418965101242065, -2.386322259902954, -1.7934778928756714, -0.14941753447055817, -0.7126627564430237, -1.2264479398727417, 0.34929922223091125, 1.8194299936294556, 0.5253305435180664, -0.08653774112462997, 0.6436464786529541, 0.14079411327838898, 0.07716682553291321, -2.370296001434326, -0.5465652942657471, 1.1949580907821655, -1.479958176612854, 1.7781931161880493, 0.05514812469482422, 1.2146039009094238, -0.00484063196927309, -0.7760951519012451, 0.8894997239112854, 1.1351475715637207, -1.1379592418670654, -0.11284525692462921, -0.09366374462842941, -0.43361830711364746, -1.4868412017822266, -0.3626891076564789, -0.35569989681243896, 0.944531261920929, -0.7417367100715637, -0.7782586216926575, 0.5596305727958679, -0.7956962585449219, -0.6449232697486877, -0.10687711834907532, -1.0475704669952393, 0.3471470773220062, 1.8551299571990967, 1.022082805633545, -0.6827720403671265, 0.0701829120516777, 0.0676710233092308, 1.6156870126724243, -0.014061210677027702, -0.5526912808418274, -0.6237377524375916, 1.1192874908447266, 1.5769540071487427, 1.397044062614441, -0.43992048501968384, 0.8784056305885315, -1.4865715503692627, 0.6185539960861206, -0.5964195728302002, -0.9931484460830688, -0.13483202457427979, -0.567385733127594, -0.39002785086631775, -0.43819859623908997, 0.4195040166378021, -0.19263993203639984, 0.44823992252349854, -0.20836381614208221, 1.2461998462677002, 1.031589150428772, -0.3456529676914215, 0.4868427515029907, 0.5349037647247314, -0.5960767865180969, -0.5670053958892822, -0.3192892372608185, 1.5131856203079224, -0.6975149512290955, -0.09683039784431458, -1.4062349796295166, -0.07356466352939606, 0.45749804377555847, -2.1676199436187744, -1.0572232007980347, 0.1640876978635788, -0.43498852849006653, -0.30082568526268005, -1.668176531791687, -0.000458975147921592, -0.29552578926086426, 1.254888892173767, 0.10967271029949188, -0.8236470818519592, 1.0685352087020874, 0.04620327800512314, -0.8455116748809814, -1.2347328662872314, -0.47849932312965393, -0.2379806786775589, 0.7298632264137268, -0.8737288117408752, -0.1242619976401329, 0.5501748919487, -0.6463646292686462, 0.34631651639938354, 0.40724247694015503, -0.9423282742500305, 1.247728943824768, 1.7034953832626343, -1.4727764129638672, 1.1751880645751953, 0.05715706944465637, 1.0614668130874634, 2.2969396114349365, 1.4137437343597412, 2.313361167907715, -1.2649667263031006, 0.06809951364994049, 0.8681362271308899, -0.3832319676876068, -0.31807148456573486, -1.5715876817703247, 1.9677579402923584, 0.5687660574913025, 0.2953222692012787, 1.157628059387207, 0.12016253918409348, 0.42215874791145325, -1.1981884241104126, 2.0589730739593506, 1.204930067062378, 0.22225798666477203, -0.508945882320404, 0.696610689163208, -2.0519182682037354, -0.19637304544448853, -1.4887597560882568, 0.6378533840179443, 1.1667474508285522, 0.6703546047210693, -0.879389762878418, -0.8737802505493164, -0.679184079170227, 1.1162256002426147, -1.9387651681900024, -0.5837222933769226, 0.38589632511138916, -0.2985019087791443, 1.1226117610931396, -0.582688570022583, -0.9240303039550781, -0.2851294279098511, 0.7414653897285461, -1.5307857990264893, -0.7975255250930786, -1.1877142190933228, 0.6897751092910767, 1.8500463962554932, -0.9812127351760864, 0.2616146206855774, -0.8912147283554077, 1.1492000818252563, -0.710793137550354, -1.7589507102966309, -0.6755549311637878, 0.690395176410675, 0.6570070385932922, 0.9161185026168823, -2.0772035121917725, -1.477195382118225, 2.1869313716888428, -0.26142293214797974, -0.24511051177978516, -0.8717895746231079, -0.6588001251220703, 2.290518045425415, 0.09736479073762894, -1.0922996997833252, -0.6472580432891846, -0.05327516794204712, 0.17020438611507416, -1.4129194021224976, -0.023473139852285385, 0.35426491498947144, 0.4236774742603302, 0.15518435835838318, 1.0148156881332397, -1.095373272895813, 0.7800394892692566, -1.8556729555130005, 1.4446170330047607, -0.8092586398124695, 0.7635740041732788, -0.22065666317939758, -0.2164492905139923, 0.2803462743759155, -0.9532260298728943, 2.387430429458618, -0.257110059261322, 0.9259656667709351, 0.06532282382249832, 0.719788134098053, 0.370849609375, 1.446083664894104, -1.592393159866333, -0.15935677289962769, -0.9678202271461487, -1.000807762145996, -1.5184321403503418, -0.7081740498542786, 0.6267452239990234, -0.005569361615926027, 0.7824184894561768, 1.0287408828735352, 0.6603057384490967, -1.0441399812698364, -0.13607294857501984, -1.0411534309387207, 0.4821731150150299, 0.5200746059417725, 0.06480883061885834, 0.28892046213150024, -0.13062027096748352, 0.03196628764271736, 1.58217453956604, -0.7167472243309021, -0.6336066722869873, -1.9793589115142822, 0.19892753660678864, 0.6652745604515076, -0.5110624432563782, -1.1960151195526123, -0.032420381903648376, 1.43618905544281, 1.0752062797546387, 0.09169948846101761, -0.3203657567501068, -0.808517336845398, -0.7232618927955627, -1.1044179201126099, -0.0910620465874672, -0.545332133769989, -1.3572173118591309, -1.5812009572982788, 1.0983237028121948, 0.7711886763572693, -1.4252605438232422, 1.5977941751480103, 1.349442481994629, 0.5095518827438354, -2.100863456726074, 0.5416331887245178, -1.0337668657302856, -1.1358505487442017, 0.02491198293864727, 0.14105114340782166, -0.12224830687046051, -0.0012568766251206398, 0.2072572112083435, -0.7157158851623535, -0.19799941778182983, -0.30945470929145813, -1.3112586736679077, 0.9443464875221252, -0.784905195236206, -0.6763861775398254, -0.6911396980285645, -0.9966574311256409, 0.6739523410797119, 1.4646083116531372, 1.9590812921524048, -0.6284533739089966, 0.8860633373260498, 0.21905024349689484, 0.22683075070381165, 3.0587966442108154, 0.40938490629196167, -0.6161206960678101, 1.0269838571548462, -1.7024002075195312, 0.2409083992242813, 0.5154826641082764, -0.20026154816150665, -1.7439234256744385, 0.6254193782806396, -1.509785771369934, -0.032564498484134674, -0.09055172652006149, 1.259173035621643, 0.9223020672798157, 0.07102438807487488, -0.47844961285591125, 0.24296444654464722, 0.2553733289241791, 0.7702766060829163, 1.501721978187561, -1.1523483991622925, -0.027226237580180168, 0.5958324074745178, -0.6048616766929626, 0.32981348037719727, 1.105239748954773, 0.4239633083343506, 0.17056798934936523, -1.2553893327713013, -0.6527519226074219, -1.8248934745788574, 0.08455076813697815, 0.9624837636947632, 0.3720950782299042, -1.7296396493911743, -0.32849135994911194, -0.5841693878173828, 0.5723276138305664, 0.8398045897483826, 0.5315430164337158, 1.6749504804611206, -0.6417040228843689, 0.6818268299102783, -0.5820322036743164, -1.1652132272720337, -1.1479586362838745, -1.2127561569213867, -1.7169206142425537, 0.24525126814842224, -2.1156187057495117, -0.18526479601860046, 0.34315043687820435, 0.5050815343856812, 0.21198461949825287, 1.447424054145813, 1.5104365348815918, 0.7531559467315674, 1.0722354650497437, 0.15187060832977295, -0.7026486396789551, -0.6220040321350098, -0.35175010561943054, 0.6839203834533691, -0.377663791179657, 0.9076125025749207, -1.1858749389648438, 0.04498962685465813, -2.753988742828369, 1.4349970817565918, -0.174162358045578, 0.2820028066635132, -0.9725102186203003, -0.3091674745082855, -0.8266023993492126, -0.578948438167572, -0.14395694434642792, 0.07141014188528061, -1.2819300889968872, 0.6258432865142822, -0.4844743311405182, -1.3318244218826294, -0.45296168327331543, 0.0855904221534729, -1.1587951183319092, -0.11720278114080429, 0.9840443134307861, -1.1897540092468262, -2.1485555171966553, 1.5094202756881714, -0.9208453297615051, -0.4477218687534332, -0.20173558592796326, 1.4896748065948486, -1.1195920705795288, -0.5687685012817383, -0.13645218312740326, -1.4462130069732666, -1.8511146306991577, -0.36870014667510986, 1.6510268449783325, 0.455282598733902, 0.2537141442298889, -1.8237793445587158, -0.28105735778808594, 1.0266380310058594, 0.11427655071020126, -0.2028064876794815, 1.2649588584899902, 0.6088080406188965, -0.25935932993888855, -0.12345512956380844, -0.6377261281013489, -0.18019695580005646, -0.944525420665741, 0.06775163859128952, -0.9148144125938416, -0.07859399914741516, 0.39272186160087585, 2.5728397369384766, -0.7423710227012634, -1.5418983697891235, 0.5014241933822632, -0.03288714960217476, -0.2959301769733429, -1.9142142534255981, -0.832033097743988, 0.026849884539842606, 1.4985235929489136, 1.2214298248291016, 1.5904961824417114, -0.2638893127441406, 0.0784534364938736, 0.4277071952819824, -1.8230239152908325, 0.6859414577484131, -0.29813292622566223, -0.4363957345485687, -1.133133053779602, 0.020043082535266876, -0.5786054134368896, -0.016476118937134743, -0.019360175356268883, -0.520577073097229, -0.8278453946113586, -0.3980923891067505, 0.5567185878753662, -2.1517765522003174, 0.7547720670700073, -0.11824537813663483, 0.8706782460212708, -0.015204190276563168, 0.5367852449417114, -0.23117497563362122, -0.842218816280365, -0.39085355401039124, 0.22038055956363678, -0.8974083662033081, -0.13468120992183685, 0.12974898517131805, 0.830653965473175, -0.5473921298980713, 0.39844152331352234, -0.38508206605911255, -0.3624234199523926, 1.014998197555542, -1.0505597591400146, 0.25923657417297363, -0.9034250378608704, -0.39077356457710266, -1.7051634788513184, -0.30266374349594116, 1.066623330116272, 0.7198996543884277, -0.988960862159729, 0.2221318930387497, 1.1184676885604858, 0.6470590233802795, 2.213862895965576, -0.6432800889015198, 0.5349859595298767, -0.19933576881885529, -1.1601670980453491, 0.22403663396835327, -1.1326453685760498, 1.323266863822937, 0.6245490908622742, 0.5303438901901245, 1.8078162670135498, 0.20035015046596527, 0.5814655423164368, 0.31551089882850647, 0.7413211464881897, -0.018698111176490784, -0.6754392385482788, -0.5583998560905457, -0.13292792439460754, 0.4769982695579529, 2.008507251739502, 0.37318944931030273, 0.8597502112388611, -0.4169432818889618, 1.0456242561340332, 0.1897633820772171, 0.8891705870628357, -1.9462822675704956, -2.523163318634033, 0.11619123816490173, 0.12354470789432526, -1.3114572763442993, 0.5682799220085144, 1.9036258459091187, 0.5176314115524292, -0.6685152053833008, -1.3683249950408936, -0.8217818140983582, 0.7991182804107666, -1.1674437522888184, -1.433782696723938, -1.2471007108688354, 0.8717393279075623, 0.5210582613945007, 0.4172397553920746, 1.2085100412368774, -0.7273871898651123, -0.5784769058227539, 0.2366105318069458, 0.09593712538480759, -1.0510036945343018, -0.15827898681163788, 1.8879904747009277, 0.2221059799194336, 1.5126410722732544, 0.49800431728363037, 1.0395203828811646, 0.8183814287185669, -0.6867695450782776, 0.5096361041069031, 1.999796986579895, 1.2792034149169922, -0.08646364510059357, 0.33606454730033875, -0.24338598549365997, 0.12071279436349869, 1.2948634624481201, -0.6525877118110657, 0.33626630902290344, 0.527277410030365, -1.2148746252059937, -1.1537208557128906, -2.1078379154205322, -0.5202080011367798, -1.1279139518737793, 0.12255380302667618, -0.6690905690193176, -0.48940327763557434, 0.02605731040239334, -0.572609007358551, 0.9395753741264343, 0.19252265989780426, 0.38818714022636414, -0.06521797925233841, -2.239748954772949, 0.42876866459846497, -0.9053871035575867, -0.9428683519363403, -0.036646101623773575, 0.45562058687210083, -0.32217276096343994, -0.3739846646785736, 1.6903140544891357, 0.5135442614555359, 0.7816714644432068, 0.39514651894569397, -1.500455379486084, 0.563101589679718, 0.06209986284375191, -1.2856961488723755, -0.4904419183731079, -0.5573212504386902, 1.2616143226623535, -1.6844191551208496, 1.5000869035720825, 1.1398876905441284, 0.49019119143486023, -1.117518663406372, -0.057225827127695084, 0.33495673537254333, 0.53956139087677, 0.5344049334526062, -0.0996021032333374, 0.43349555134773254, 0.7586717009544373, -1.0914417505264282, 0.3825148046016693, -0.8718600869178772, -0.0619453601539135, 1.027087926864624, -1.277244210243225, -0.24113011360168457, -2.0622119903564453, -0.437004953622818, 1.9374125003814697, -1.6124937534332275, -0.09913807362318039, 1.1107773780822754, -0.44982069730758667, -1.079420804977417, -0.24268434941768646, -0.5094570517539978, 1.1207518577575684, 0.0596151277422905, 0.23500046133995056, 0.6206182837486267, -0.32020828127861023, -1.2374458312988281, 0.2899773120880127, 0.8982537984848022, 0.3241865038871765, 0.7181901931762695, 0.5491349101066589, -0.1584189236164093, 0.9167441129684448, 0.7496577501296997, 1.6103023290634155, 0.10303159058094025, -2.008854627609253, -0.12480911612510681, 0.3446751534938812, -1.1525362730026245, -1.0200241804122925, -1.7932701110839844, 0.16569827497005463, -0.11763256788253784, 1.0672860145568848, -1.850974440574646, 2.0329558849334717, 1.5039178133010864, 0.9684886932373047, 0.5828462243080139, 0.3234090209007263, -0.8303031325340271, 0.15351352095603943, 1.4822577238082886, -0.6567175984382629, -0.11475234478712082, 0.5309196710586548, -1.1282094717025757, -0.14859449863433838, 0.5810374617576599, -0.3250628709793091, 1.185263991355896, 0.5618388056755066, -0.6651855707168579, -0.09520253539085388, -0.4184381067752838, 1.2491823434829712, -0.5082768797874451, -0.2883293032646179, -0.8620936870574951, -0.29411178827285767, -0.324470192193985, -0.1922421008348465, -0.12465649098157883, 1.8212794065475464, 0.6656845808029175, 0.5944756269454956, 2.8422892093658447, -0.17572331428527832, 2.1167447566986084, -0.01885306090116501, 0.6543919444084167, -1.1523102521896362, -0.7224080562591553, -2.1447832584381104, -1.1268103122711182, 0.07199711352586746, -1.5498758554458618, -1.195480465888977, 0.17905065417289734, 0.08736859261989594, 0.16919051110744476, 0.3913719058036804, -0.21279063820838928, 1.753532886505127, 0.3480229675769806, -0.12727737426757812, -1.8523138761520386, 0.23029668629169464, -1.0756324529647827, 0.9494770765304565, -0.4784581959247589, -1.520628809928894, -1.1594313383102417, 0.9335576891899109, -0.29361289739608765, 1.5504049062728882, -1.2826638221740723, 0.8764474391937256, 0.7023751735687256, -0.9841339588165283, 0.006495485547930002, 0.9084365367889404, -0.7676500678062439, -0.47864973545074463, 0.46147194504737854, 1.5925410985946655, -0.6751531958580017, -0.46705859899520874, 0.01286875456571579, 0.4869473874568939, -0.2716308832168579, 1.1033861637115479, 1.1087673902511597, 1.114685297012329, -0.07804755866527557, 1.365169644355774, 0.7467129826545715, 0.5089071989059448, 0.207443967461586, 0.22535809874534607, -1.0582101345062256, -0.1457633078098297, -1.4280225038528442, -0.055066950619220734, 0.5609579682350159, 0.5890226364135742, 0.8414255380630493, 0.8135776519775391, 0.7838574051856995, -0.4182904064655304, 1.308660626411438, -1.3077237606048584, -0.07339999824762344, 0.06380750238895416, 0.34660717844963074, 0.7549297213554382, 1.174274206161499, -1.9007830619812012, 1.2946257591247559, -1.977183222770691, 1.5501261949539185, -0.2871038019657135, -0.9229021072387695, 0.34027838706970215, -1.1838699579238892, -0.07797402888536453, -0.2710890471935272, 0.2417343109846115, -1.9778589010238647, -1.371053695678711, -0.411415159702301, -0.5392288565635681, 1.4046027660369873, 1.7865818738937378, 0.24369092285633087, 0.5037996172904968, -1.2557789087295532, -1.1436375379562378, -1.9780831336975098, 0.5450609922409058, 1.7173422574996948, 1.2347034215927124, -0.6159545183181763, 0.3540092408657074, -0.6009626984596252, 0.7598584294319153, -0.9144670367240906, -0.6151651740074158, 0.5577660202980042, -0.9969502687454224, 2.249241828918457, 0.7632557153701782, -1.737730860710144, 0.06706555932760239, -0.5042831897735596, 0.6193366050720215, -0.6329454183578491, -0.70931476354599, 1.93893563747406, 0.3208479583263397, 2.1788856983184814, -0.8670551180839539, -1.2238703966140747, 0.49204468727111816, 0.2427329123020172, -0.32991334795951843, -0.3736391067504883, 0.4898785352706909, 1.3165422677993774, -1.386748194694519, 0.8956311941146851, 0.28401222825050354, 0.0451783649623394, 0.8515340089797974, 1.6301835775375366, -0.01579052209854126, 0.9867995977401733, 1.2188514471054077, 0.035184748470783234, -0.7356388568878174, 1.532140851020813, 0.03554004430770874, 0.8943925499916077, -0.2762586176395416, 0.03678886964917183, 1.8330012559890747, -2.0135772228240967, 0.6481293439865112, -0.2342662364244461, -0.6570871472358704, -0.9541114568710327, -0.8390592336654663, 0.5394046306610107, 0.08970267325639725, 1.703292727470398, 0.4248422086238861, 0.951612114906311, 0.27220726013183594, -0.18381193280220032, 1.0307568311691284, 0.7769407033920288, -0.15825076401233673, -0.2867153286933899, -0.7564301490783691, -0.8651493191719055, 0.6936690211296082, 0.8293517827987671, 2.169877290725708, -0.41299694776535034, 0.1267876923084259, -0.15904749929904938, 0.33235448598861694, 0.4286612868309021, -0.009060355834662914, 0.6712833046913147, -0.2115131914615631, 0.5477131009101868, 1.558415412902832, -1.1523274183273315, -0.02523081935942173, -0.06234751641750336, -0.7221498489379883, 1.4487560987472534, -0.8015563488006592, -0.34417474269866943, 0.06616346538066864, 0.19833262264728546, -0.01577947847545147, -0.8181403875350952, -0.260562539100647, -0.5409128069877625, -0.33629393577575684, 0.29500776529312134, 2.2121598720550537, 0.30216747522354126, -0.5222676396369934, -1.5160176753997803, -0.9819807410240173, -0.5277012586593628, -2.047926664352417, -0.6484479308128357, 0.341968297958374, 1.9028546810150146, 0.09874892234802246, 1.6787290573120117, 0.4914018511772156, 0.8131659626960754, 2.299276351928711, -0.1280088722705841, -0.8605544567108154, 0.5294193029403687, -0.16406363248825073, -1.8608397245407104, -1.3579494953155518, -0.7501096725463867, 0.8638150095939636, 1.203923225402832, -1.80715012550354, 0.34099119901657104, 0.5191522836685181, 2.124746084213257, 0.9069144129753113, 0.2906792163848877, 1.2607094049453735, -2.046879768371582, 0.14804697036743164, 2.3538057804107666, 0.5816148519515991, 0.9718465805053711, -0.8186111450195312, -0.660651445388794, 1.029853105545044, -0.5830193758010864, -0.36546123027801514, 0.3396683633327484, -0.5397685170173645, -1.8743116855621338, 1.239729881286621, -3.663435220718384, 1.1555026769638062, -0.43229252099990845, -0.6883644461631775, 0.8705728650093079, 0.16297267377376556, -0.23148246109485626, -0.948590874671936, 0.18502087891101837, 1.1013481616973877, -0.5747518539428711, -1.0721733570098877, 0.9773319959640503, -0.3652936816215515, 3.3996639251708984, -1.1316337585449219, 0.508460521697998, 0.6611056923866272, 0.0013235188089311123, 0.8008820414543152, 2.128809928894043, 2.483949899673462, -1.023626685142517, 0.48380088806152344, 0.041581425815820694, -1.9600236415863037, -1.1221450567245483, 1.6412330865859985, 0.26138943433761597, 0.8588553667068481, -0.6914251446723938, 0.283282995223999, 0.21826224029064178, 1.1340646743774414, -0.7049616575241089, 0.6078200340270996, 0.19648806750774384, 1.2294880151748657, 0.029259633272886276, 2.4715991020202637, -0.7950618863105774, -1.732305884361267, -1.1655353307724, 0.20365767180919647, 0.312979131937027, 0.5243892073631287, 0.4012487232685089, -1.1418766975402832, 0.6708371043205261, -0.34869909286499023, -1.7725909948349, 0.790919840335846, 1.6646353006362915, 1.0603452920913696, 0.1304612010717392, 0.5481629371643066, 0.4103809595108032, -0.4533599615097046, -0.08358094096183777, 1.2887033224105835, 1.1955575942993164, 1.2238489389419556, -1.284288763999939, -0.6120355129241943, -0.17603859305381775, 1.5095386505126953, -0.649208128452301, -0.3947999179363251, 0.0969279333949089, 1.0631215572357178, 0.8818715810775757, 0.7721918225288391, -0.3042708933353424, 1.8168004751205444, -0.7713615894317627, 2.0644593238830566, 0.24745866656303406, -0.43014541268348694, 0.05282875522971153, 0.7315551042556763, -0.41808751225471497, 0.5146359801292419, -0.671408474445343, 0.8301615715026855, -0.465588241815567, -0.4154522716999054, -0.857005774974823, -1.1774020195007324, 0.8856713175773621, -0.8623563051223755, -0.6882859468460083, -0.3399133086204529, 0.516974925994873, -1.2885806560516357, -0.9749781489372253, -0.35919398069381714, -1.1070668697357178, -1.3756531476974487, -0.2787352204322815, 0.2344275265932083, -0.635532021522522, 0.6422123908996582, -0.8546015024185181, 0.3807829022407532, 0.7942524552345276, -2.154228687286377, -0.39250320196151733, -0.7084792852401733, 1.3533718585968018, 1.0971474647521973, 0.0776679739356041, -0.9759072065353394, -0.3851809799671173, -0.4757312834262848, 0.10016129910945892, -0.8007164597511292, -1.229821801185608, -0.17455017566680908, -0.6310304999351501, 0.46245765686035156, 0.5970364809036255, -0.37063395977020264, 0.9184010624885559, -0.0275510773062706, 0.8618541359901428, -0.12172411382198334, -2.1358275413513184, -0.7226569652557373, -0.8402600288391113, -0.886627733707428, 0.29708459973335266, 0.28307226300239563, -0.9303833842277527, 0.35890284180641174, -0.25648173689842224, -0.5737788677215576, 0.828805685043335, 0.6395288109779358, 1.2217557430267334, -0.14922116696834564, -0.7414480447769165, -0.9811001420021057, -0.014055522158741951, 1.619864583015442, 1.2397527694702148, 0.8513826727867126, -0.9726642966270447, -1.3764159679412842, 1.4871065616607666, -0.05138777941465378, -0.7458540797233582, -0.16480663418769836, -0.9356511235237122, 0.3204785883426666, 1.050655484199524, -0.8598148226737976, -1.2851848602294922, -0.6608229875564575, 0.2160687893629074, -0.09296058863401413, -0.18858622014522552, -1.4073582887649536, -0.21296539902687073, -2.0651795864105225, 0.9450510740280151, 0.6690006852149963, 1.577032208442688, 0.7459102272987366, 0.5467342734336853, -0.25018006563186646, -0.9697917699813843, -1.0597736835479736, 0.0918274000287056, -1.2483315467834473, -0.8108563423156738, 0.4405950605869293, 0.38207346200942993, 1.6296007633209229, -0.3027873933315277, 0.5740460753440857, 0.07747811824083328, 1.7581263780593872, -1.921812891960144, 1.4107881784439087, 0.38236507773399353, 1.141900897026062, -0.5289482474327087, 0.766136884689331, 0.37709444761276245, 0.3031109571456909, -0.7322115302085876, 0.18234588205814362, -0.35768988728523254, 0.1558942198753357, 1.1289700269699097, 1.2037386894226074, 0.0425855927169323, -0.06842000782489777, -2.2902262210845947, 1.3018947839736938, -0.6089770793914795, -0.6324528455734253, -1.804609775543213, -1.3521212339401245, -0.09222287684679031, -0.5440793037414551, 0.22857649624347687, 1.7375819683074951, 0.7587169408798218, 0.3481391966342926, -1.3205646276474, -0.8801670074462891, 0.3344459533691406, -0.9628199338912964, 1.2286393642425537, -0.14544886350631714, 1.1567082405090332, 0.9523298740386963, -0.41019734740257263, -1.7457951307296753, 1.2055917978286743, 0.8433561325073242, -1.5275870561599731, -0.1557529866695404, -0.18364034593105316, 0.31861644983291626, -0.06806206703186035, 0.13581469655036926, 0.5929733514785767, -1.7860777378082275, -1.0670503377914429, 0.3945521116256714, -1.6035150289535522, 0.636105477809906, 0.021517718210816383, 1.7124637365341187, 1.0039132833480835, 1.563717007637024, -1.0198078155517578, 0.38140395283699036, -1.4613687992095947, 0.10911928862333298, -3.488563060760498, -1.6254996061325073, -0.5959121584892273, 0.508756697177887, 0.6690534353256226, -2.152308702468872, 0.13368633389472961, 0.6325392723083496, 0.12686502933502197, 0.5968279838562012, 0.912693202495575, 0.5725497007369995, 0.04142335429787636, 0.23648175597190857, 1.2706267833709717, 0.562926709651947, -0.6327964067459106, -0.015374260023236275, 0.3157770335674286, 0.1461641490459442, -0.21682609617710114, -1.221437692642212, 0.48749715089797974, 0.76256263256073, 0.9322375059127808, 1.2034687995910645, 1.408745527267456, -0.32234132289886475, -0.14815132319927216, -1.3029183149337769, -1.2412039041519165, 2.084613561630249, 0.1796063929796219, -1.602668285369873, -0.8511278629302979, 0.8659405708312988, 0.3791755139827728, -0.4779432415962219, -0.3489331007003784, -0.44461360573768616, -0.2760752737522125, 1.4627152681350708, -0.6347686052322388, 0.34619906544685364, 0.716335654258728, -0.21330954134464264, 2.1760168075561523, 0.21009056270122528, 0.9788104891777039, -1.7095001935958862, 1.4323734045028687, -0.20267046988010406, -0.03761022537946701, 0.6805681586265564, 0.14448879659175873, -1.1326093673706055, -0.8570088148117065, -1.7430427074432373, -0.9307659268379211, -1.6238435506820679, -0.9156660437583923, 0.5220640897750854, -0.7020201683044434, 0.318633109331131, 1.9608442783355713, 0.29583972692489624, 0.22555623948574066, -1.1500303745269775, 0.08767686784267426, -0.5994256734848022, 0.39422571659088135, -0.6124434471130371, 1.0305272340774536, 1.0610038042068481, 1.0346981287002563, -1.8334827423095703, 0.5696923732757568, 0.6018456816673279, -1.3805512189865112, 0.21005116403102875, 0.25543537735939026, 0.21435213088989258, -0.41551336646080017, -0.6962095499038696, -0.5378161072731018, 0.7008653879165649, -0.5685720443725586, -0.33076024055480957, 0.9952201843261719, 0.7932456135749817, -0.2328716218471527, 0.896069347858429, 2.013335704803467, -0.14244329929351807, -0.39908313751220703, -0.3088364899158478, -0.05215312913060188, -1.0264045000076294, 1.456544280052185, -0.11727162450551987, 0.5219037532806396, 1.4454227685928345, 0.6019774079322815, 0.26832330226898193, -0.7854686975479126, -0.7664692997932434, -1.1062865257263184, 0.42996323108673096, 1.358734130859375, -0.45738765597343445, -1.2248201370239258, -1.0463050603866577, 1.2744861841201782, 0.6783187985420227]], "dtype": "float32"}, {"__nd__": [[-1.37901771068573, -1.581894040107727, -1.0101503133773804, -0.262798011302948, -0.3572462797164917, 1.042965292930603, 0.8956244587898254, 1.7192275524139404, 0.42691612243652344, -0.5307531952857971, 0.882830023765564, 0.6204129457473755, -1.5698859691619873, -0.1654737889766693, 0.26393553614616394, 0.6968348622322083, 0.7385796308517456, -0.889168381690979, -0.11698479950428009, 0.6256411075592041, -1.4951400756835938, -1.4027490615844727, 0.22052061557769775, -1.1899778842926025, 0.6444151997566223, 0.02241654135286808, -0.34852612018585205, -0.1669793426990509, 1.4652652740478516, 1.8969193696975708, 0.6591615080833435, 0.771393895149231, 0.6700373291969299, 1.9369347095489502, 0.7926595211029053, -1.067134976387024, 0.1817464679479599, -0.5210910439491272, -1.4327400922775269, -0.1103207990527153, -0.09069942682981491, 1.4996672868728638, -0.09027441591024399, -0.215791717171669, 0.05309798941016197, 0.08758388459682465, 1.2407463788986206, 0.1356482356786728, -0.76301109790802, -0.5591439604759216, -0.16377957165241241, 0.3810434639453888, -1.7346820831298828, -0.36055296659469604, 0.23808975517749786, -0.21732325851917267, 1.2588859796524048, -0.4991762042045593, -0.49094894528388977, 0.6273399591445923, 0.5022244453430176, 0.10894551873207092, -0.5127748250961304, 0.4034525752067566, 0.6714978814125061, -1.1342519521713257, -0.26729342341423035, -0.11188904941082001, 0.5995451211929321, -0.18150575459003448, -1.3376307487487793, -0.6490969061851501, 0.21938006579875946, -0.5712565183639526, 0.046412721276283264, 1.2068997621536255, -0.41919195652008057, 0.483417809009552, 0.03373179957270622, 0.34247884154319763, 1.2756950855255127, -0.3834768533706665, -0.16733543574810028, -1.573664903640747, 2.245054006576538, 0.4908570349216461, -1.5377954244613647, 0.5710841417312622, -0.14264298975467682, -2.005896806716919, 0.4827911853790283, -0.466007262468338, 0.19389860332012177, -0.38412532210350037, -0.10680986940860748, 0.6838629245758057, 1.1147089004516602, -0.5378730893135071, -0.6232845187187195, -0.3748764097690582, -0.18803371489048004, 0.6101462244987488, 1.2773587703704834, -2.2096502780914307, -1.481091856956482, 0.9069616198539734, 0.6458537578582764, 1.762351632118225, -0.2815488278865814, 0.7239090204238892, -2.107067108154297, 1.0940823554992676, 0.5634880065917969, -0.8541127443313599, -0.6724807620048523, -0.7186823487281799, 0.055810414254665375, 1.4116555452346802, -0.15169034898281097, 0.9744747281074524, -0.40232276916503906, 1.1373076438903809, -1.6863377094268799, -0.33626702427864075, 0.597430408000946, 0.16420774161815643, -1.3807975053787231, 0.7801021933555603, -0.2737329602241516, -0.7786167860031128, 0.7931581735610962, 0.10832918435335159, -0.13323913514614105, 0.6443074345588684, 0.37735119462013245, 0.9472364187240601, 1.3534260988235474, -0.3193199038505554, -0.2570398151874542, 0.7245341539382935, -1.7120782136917114, -1.3095072507858276, -1.1118943691253662, -1.2831590175628662, -0.5356055498123169, -0.885412335395813, -0.6414216160774231, 0.15048028528690338, 0.32588839530944824, 1.4928337335586548, -0.8736806511878967, -0.13173650205135345, -0.10177318006753922, 0.8921354413032532, 0.015612971968948841, 1.488703966140747, 0.35318881273269653, 0.6767394542694092, -0.784730076789856, 0.09196467697620392, -0.577505886554718, 0.02553914673626423, 1.6985068321228027, -0.21886877715587616, -0.2619972229003906, -0.16544559597969055, 0.24693578481674194, 2.656114339828491, -1.8091061115264893, -0.6948792934417725, 0.40647828578948975, 2.306994915008545, 1.2399117946624756, 0.6433746218681335, -1.0758765935897827, 0.670339047908783, 0.5265371799468994, 0.10353320837020874, 1.363553762435913, -0.7377407550811768, -0.8680461645126343, 0.6838671565055847, 0.24458619952201843, 0.03330332040786743, 1.230613112449646, 0.20199836790561676, 0.8234579563140869, -0.5219626426696777, -1.444159984588623, -1.1941559314727783, -0.36585375666618347, 0.5806864500045776, 0.6981214284896851, -0.42825740575790405, 0.5588656067848206, 0.6504753828048706, 0.28285545110702515, -0.3986494839191437, -0.9714895486831665, 1.1131412982940674, -0.9713371396064758, 0.18621888756752014, 0.08882607519626617, 1.2427390813827515, 0.8146870136260986, -1.7949681282043457, -0.5750757455825806, -0.6519562602043152, -0.49040767550468445, 0.8494340777397156, 0.8626537322998047, -0.39634576439857483, -2.3712151050567627, 0.0078062573447823524, 0.1650521159172058, -0.8690041303634644, 0.21762801706790924, 1.2254869937896729, -1.5064469575881958, 1.407138705253601, -0.9004756808280945, -1.0049084424972534, 1.0989809036254883, -0.48094287514686584, -0.07063209265470505, 1.4679220914840698, 0.3153096139431, -0.03519536554813385, -2.299118995666504, -0.7261022925376892, 0.5362431406974792, 1.5415323972702026, 0.8141851425170898, -0.002299713436514139, -1.580702781677246, -1.3913475275039673, 0.7799155116081238, -0.6048325896263123, 0.635786771774292, 1.5707004070281982, -0.7660084962844849, -0.06843272596597672, -0.26128995418548584, -0.5160800814628601, -0.9941498637199402, 1.1501884460449219, 0.11991576105356216, 0.08442338556051254, 0.6201698184013367, 0.3339812159538269, -0.16655954718589783, 0.4732794165611267, 2.7505154609680176, -0.7269707322120667, 0.7570191621780396, -2.1141083240509033, 0.8501027226448059, 0.7937332987785339, -0.40587717294692993, -0.9486197233200073, 0.22580625116825104, 0.5969972014427185, -0.9833894968032837, 1.3756232261657715, 0.8460224270820618, 0.23021049797534943, 0.23349405825138092, -0.8766602873802185, -2.205461263656616, -0.5349118709564209, 0.9949935078620911, 0.43224191665649414, -0.9902381300926208, -1.5166277885437012, -1.3490617275238037, -0.5597916841506958, 0.220199316740036, 1.4474846124649048, -0.1340591162443161, 1.3519418239593506, -1.5052523612976074, -0.34249553084373474, 1.1129578351974487, 0.97060227394104, -1.4638686180114746, -0.8204700946807861, 0.39158686995506287, -0.8300517201423645, -1.0727167129516602, 0.31635725498199463, -0.6632668972015381, 1.2452059984207153, 0.5482382774353027, 1.0380114316940308, 0.3941023647785187, -0.014910115860402584, -0.9234539270401001, 0.026851270347833633, -1.1865026950836182, -1.9761375188827515, -1.000963568687439, -2.385420560836792, -0.5546857118606567, 1.8953016996383667, 0.09220247715711594, 0.06268385052680969, -1.203834056854248, 0.026450203731656075, -0.8981626033782959, 0.1540631800889969, -0.5891799926757812, 0.6098425984382629, -0.6265603303909302, -0.945278525352478, -1.7015907764434814, 0.2987750470638275, 0.8226641416549683, -1.3127621412277222, 1.2097163200378418, -1.4547765254974365, 1.613891363143921, -0.2060353010892868, 1.252447247505188, -0.722349226474762, 0.19805288314819336, 0.24197322130203247, 0.2698204517364502, 0.08802270144224167, -1.2202790975570679, 0.0813620314002037, -1.3973190784454346, 0.7294901013374329, -0.05183856561779976, 0.3635839521884918, 0.3979825973510742, 0.5356864929199219, -0.7382996082305908, -0.5238083600997925, 0.7972266674041748, 1.994542121887207, -1.3111283779144287, -0.3720335364341736, 1.1259576082229614, 0.557580292224884, 1.0916558504104614, 3.6258904933929443, 0.1790620982646942, -0.380695641040802, 1.230453610420227, -1.853821873664856, -1.568334698677063, 0.44091978669166565, 1.0768026113510132, 1.0571738481521606, 1.8338030576705933, 0.13067086040973663, -0.9304790496826172, -0.8367904424667358, -0.6071298122406006, 0.03219439834356308, 0.7837528586387634, 0.5720712542533875, 0.6823760867118835, 0.05781150236725807, -0.691135585308075, -1.2196996212005615, -0.4517594873905182, -1.4284347295761108, -0.2691287696361542, -0.3201235830783844, -0.0867650955915451, 0.8768330812454224, 0.7897236347198486, -0.4846607744693756, -1.0083101987838745, 0.33625689148902893, -1.0124672651290894, -2.129507303237915, -0.2674098312854767, 0.5365221500396729, -0.1004168912768364, -0.5080202221870422, 2.242072343826294, 1.7658727169036865, 0.007554605603218079, 0.001792383729480207, -0.06381218880414963, 2.1587421894073486, 0.4174545705318451, -0.9452450275421143, 0.43003252148628235, -0.6010690331459045, -0.6604428291320801, 0.20388029515743256, 1.889857292175293, -1.1869118213653564, -0.330552339553833, 1.515509009361267, -0.6101104021072388, -0.621789813041687, -0.29073405265808105, 0.09378428012132645, 0.1320570856332779, -1.0668598413467407, -0.8888223767280579, 0.4356362819671631, -0.9776818752288818, -0.1621691882610321, 0.13646546006202698, -0.932871401309967, 0.040549807250499725, 0.9348934888839722, 0.9600313305854797, -0.632377564907074, -1.6575956344604492, 0.48416852951049805, -0.08493825048208237, -0.9944915175437927, 0.47631317377090454, -0.29301419854164124, -0.7976896166801453, 0.25605249404907227, 0.48264431953430176, 1.1664296388626099, 0.7041112184524536, 0.15068615972995758, 1.1222811937332153, 0.004767184611409903, -0.2955387234687805, 0.24552440643310547, 1.2238534688949585, -0.3533598780632019, -1.7458847761154175, -0.016392990946769714, 0.2704458236694336, 0.294861763715744, -0.1528339982032776, -0.3781193792819977, -0.27193981409072876, -0.1294824182987213, -1.5966582298278809, 0.63450026512146, 0.5198389291763306, -1.5947710275650024, -0.9410781264305115, -0.43730637431144714, 1.3397712707519531, 1.0944205522537231, -0.7785578966140747, -0.917733371257782, -1.4984486103057861, 2.449680805206299, 0.06807775795459747, 0.1965002417564392, 0.47590523958206177, 0.8654459714889526, 0.778090238571167, -1.5549687147140503, 0.18194961547851562, -1.5579280853271484, 0.43272337317466736, -0.9624432325363159, -1.205918550491333, 1.3128584623336792, 2.0873100757598877, 1.1854004859924316, 0.2802239656448364, -0.026028525084257126, -0.3518245220184326, 1.46570885181427, -0.25378355383872986, 0.780828058719635, -1.0246251821517944, 0.5487337112426758, 0.625469446182251, -0.43388745188713074, 1.2168817520141602, -0.34444546699523926, 0.18417119979858398, 2.3870303630828857, 0.31311067938804626, 2.3281068801879883, 0.4827185571193695, 0.21617868542671204, 0.15249590575695038, 2.7465457916259766, -0.5255656242370605, 0.6904684901237488, 0.6460906267166138, -0.5815111994743347, -0.8300760984420776, -0.637560248374939, 0.5492370128631592, 0.7992208003997803, -0.9260095953941345, -0.5712098479270935, 0.8507535457611084, -0.3568441867828369, -1.6270620822906494, -0.7022570371627808, -1.8227460384368896, 0.6516301035881042, -1.2575793266296387, -0.3148297667503357, -1.3527003526687622, 0.04890337213873863, 0.4995998740196228, -0.34196075797080994, -0.8102198243141174, -1.7512588500976562, 0.68059903383255, 1.2195340394973755, 0.2853304147720337, 1.3184449672698975, -1.4523611068725586, 1.3489435911178589, 0.8996146321296692, -1.4617093801498413, 0.6095547676086426, -1.2820556163787842, -1.1297056674957275, 0.5395038723945618, 0.9915993213653564, -0.36846911907196045, -0.380628764629364, -0.585109293460846, -0.9490693211555481, -0.5637436509132385, 1.856824278831482, 0.2825678288936615, 0.902435839176178, 0.5813419222831726, -0.5918523669242859, -1.233176589012146, 1.031921148300171, 0.12374989688396454, -0.015030942857265472, 0.6212735772132874, -0.9903043508529663, 0.8711163401603699, 1.049239993095398, 0.3014092743396759, -1.8570926189422607, -0.6005163192749023, 0.5481534600257874, -0.9638146758079529, 0.935882031917572, -1.4160784482955933, -0.9670756459236145, -2.5500385761260986, 0.6903890371322632, 0.6135721206665039, -0.2809017300605774, -0.15580037236213684, -2.3889710903167725, 1.9766143560409546, 1.743695616722107, -0.2076837420463562, 0.49212539196014404, -0.03261943906545639, -1.1292777061462402, 0.9260947704315186, 0.019170744344592094, 0.5442391037940979, 1.2241898775100708, 0.971310555934906, -1.4608224630355835, 0.9985963702201843, 1.0197243690490723, 0.97822505235672, -0.13552848994731903, -0.8880148530006409, -0.1879362314939499, -0.3303149938583374, 0.2915581166744232, -0.7041124105453491, 1.3768147230148315, -0.09520170837640762, 1.6546130180358887, -0.4763294458389282, 1.9213283061981201, 1.5479532480239868, 0.26643598079681396, -1.7527458667755127, 0.5252392292022705, 1.8537453413009644, -0.7932199239730835, 0.6845026016235352, 1.5091819763183594, -0.6001802086830139, 0.22606366872787476, 0.16111481189727783, 0.7867168188095093, 0.00925414077937603, -0.6009082794189453, 2.198272943496704, -0.4581240117549896, 0.49076342582702637, 0.6658249497413635, 2.0180275440216064, -0.9906988739967346, 1.339707612991333, 0.5917322039604187, -0.4413715898990631, 1.9057594537734985, -0.22147159278392792, -0.5094695687294006, -1.0221216678619385, -0.7624210715293884, 0.7886781692504883, -0.5183382034301758, -0.9625428915023804, -1.1787786483764648, 0.11303749680519104, -1.8713018894195557, 0.4962429702281952, -0.7267802953720093, 0.2502351999282837, 1.2785624265670776, -0.9882543683052063, -0.31983017921447754, -0.453049898147583, 0.8186636567115784, -1.3201707601547241, 1.3820183277130127, 0.042761776596307755, 1.9997721910476685, -0.4743110239505768, -0.2747131586074829, 2.1265878677368164, 0.5319873690605164, -2.6790850162506104, 1.903682827949524, -0.18944379687309265, -0.9236096143722534, -0.2633008062839508, 0.5931804776191711, 0.3635464310646057, 0.7460119128227234, -0.986807644367218, -1.2407124042510986, 0.6264866590499878, -0.35467734932899475, 1.6639912128448486, -1.589328646659851, 1.4413594007492065, 0.34038472175598145, -0.26212450861930847, 1.19624924659729, 1.6989413499832153, -1.0637235641479492, -0.8588131070137024, 1.8113328218460083, -1.348271131515503, -1.201859951019287, -0.3253440260887146, 1.4003123044967651, -0.4476854205131531, 0.8316044807434082, -0.5281243920326233, -0.010035393759608269, -0.5041645765304565, 2.6793153285980225, -0.5568931102752686, -0.5626509785652161, 1.4269123077392578, 2.4523122310638428, 1.5118482112884521, -0.7196127772331238, -0.3601847290992737, -0.33094629645347595, -1.2208713293075562, 0.20735901594161987, 0.0991843119263649, -0.2308119386434555, -0.551986038684845, -0.12310008704662323, 0.46645864844322205, 0.013140229508280754, -0.7330962419509888, 0.5001953840255737, 0.3273756504058838, 0.07590825110673904, -0.8681603670120239, 0.8565477132797241, 1.2011903524398804, 0.627508282661438, 0.8142824769020081, 0.8559651970863342, 0.4698215425014496, -0.34286293387413025, -0.6027312278747559, 0.42337846755981445, -1.328116774559021, -0.9450412392616272, -0.8134448528289795, 0.2293812334537506, 0.6901359558105469, 0.8732867240905762, 0.6684488654136658, 0.7066860198974609, -1.1804590225219727, -1.1331979036331177, 0.20016886293888092, 1.8046200275421143, 0.16713528335094452, -0.9834719896316528, -0.8376818299293518, 0.32030338048934937, 0.3032011389732361, -1.1401089429855347, 0.36993083357810974, 0.5697457790374756, -2.046727180480957, 2.077327251434326, 0.7906930446624756, 0.9873537421226501, -0.7491164803504944, 0.17371225357055664, -1.0709521770477295, -0.013158421963453293, 1.794898271560669, 0.9154353141784668, -0.10714758932590485, 0.9096369743347168, 0.701914370059967, 0.2898271381855011, -0.09919299930334091, -2.089491844177246, -1.4215830564498901, 1.6232264041900635, -1.7637540102005005, 1.830321192741394, 0.025194501504302025, -0.5630939602851868, 1.3170853853225708, 0.15073442459106445, -0.9257817268371582, 0.8688603639602661, 0.8835732936859131, -0.6770148873329163, 0.709949791431427, 0.28312090039253235, -0.9093376398086548, 0.5621681809425354, 0.41530725359916687, 1.563023567199707, 0.014212195761501789, -0.05983465164899826, 1.6291579008102417, 0.8910653591156006, 0.25082775950431824, -1.6178690195083618, -1.6246883869171143, -2.188617706298828, -1.3077985048294067, -0.15324831008911133, -0.26414430141448975, 0.45731741189956665, 0.40883252024650574, -1.1201521158218384, 0.7133257985115051, 0.7115640640258789, 0.12097206711769104, -1.417490005493164, -0.40259239077568054, -0.8600068688392639, 0.11186755448579788, 0.45495128631591797, -0.8459075689315796, 1.572661280632019, -1.219997763633728, -1.14506995677948, 0.22349479794502258, 0.522341787815094, -1.0699504613876343, -0.10414556413888931, 0.2829664349555969, 3.208117961883545, 0.8262491822242737, -0.6878166794776917, -0.031139379367232323, 0.8103705644607544, 0.18151919543743134, -0.01918429508805275, 2.6511340141296387, -0.5274650454521179, -0.500254213809967, 0.35496029257774353, 0.46047526597976685, -1.9742677211761475, 0.5166732668876648, 1.2880852222442627, -1.0865178108215332, 1.9429455995559692, 0.361730694770813, 0.6927216649055481, 0.3537057042121887, -0.08624015003442764, 0.5319026112556458, 0.06900115311145782, -0.2226216346025467, 1.3407703638076782, -0.47603994607925415, -1.1045926809310913, 1.4718390703201294, 0.02129622921347618, 0.08480208367109299, -0.37625041604042053, 0.9631717205047607, -2.6024727821350098, 0.5769737958908081, -0.7528854012489319, 0.7078498601913452, 0.12853950262069702, -0.01346401683986187, 0.17255394160747528, -0.4861305356025696, 1.2320244312286377, -0.22318977117538452, 1.134985327720642, 0.3687249720096588, 1.4035511016845703, 0.7469407320022583, -0.0539030097424984, 0.32195380330085754, 1.042235016822815, -0.9063793420791626, 0.5214360952377319, 0.4052804410457611, 0.26330938935279846, -1.1787216663360596, 0.3524930477142334, 0.7336450219154358, -1.38468599319458, 0.18329082429409027, -0.8133454322814941, -0.7712764143943787, 0.3193475008010864, 0.12655219435691833, -1.3572101593017578, -1.287247657775879, 1.864642858505249, -0.5050703287124634, -0.6941457390785217, -0.5956059694290161, 0.0192774198949337, -0.4364711046218872, 0.05630064383149147, 1.7746208906173706, -0.6301182508468628, 0.8203253149986267, -0.721439003944397, 0.286981463432312, 0.08377330005168915, 0.421352356672287, 1.5909618139266968, -0.11662296950817108, -0.5838017463684082, 1.266735553741455, 0.2938997447490692, 0.20025739073753357, 0.6275755167007446, 1.7443422079086304, -1.423750638961792, 0.5064162611961365, -2.613846778869629, 1.1568156480789185, 0.9290058612823486, 0.8164637088775635, -0.1859724521636963, 0.5449294447898865, -1.3416521549224854, -1.0712380409240723, -0.3398725986480713, 0.5008113980293274, -0.30569344758987427, -0.5699110627174377, -0.07584945112466812, -1.292648434638977, -0.9609613418579102, 0.6683031320571899, -0.6479414701461792, -1.170441746711731, -0.10690134018659592, 2.0821828842163086, 1.0745769739151, -1.5600870847702026, 0.02375771664083004, 1.732834815979004, -1.6796739101409912, -0.47540315985679626, 1.0831035375595093, 0.16418339312076569, 0.9096715450286865, -1.4750003814697266, -1.5176076889038086, -1.7294394969940186, -0.36307594180107117, 1.061672329902649, 0.706457793712616, 0.9793379902839661, 0.439042866230011, -0.5203768610954285, 1.8244359493255615, -0.6153201460838318, -0.664047122001648, 2.2416486740112305, 1.2991739511489868, -1.7287958860397339, 0.4923052191734314, 1.4516510963439941, 2.4528908729553223, 0.31136325001716614, 0.18981343507766724, -0.7472980618476868, 0.7536646723747253, 1.0565528869628906, -0.50569748878479, 0.3380122780799866, 1.882775902748108, -0.1493413746356964, -1.0528355836868286, 0.046651218086481094, 0.36046698689460754, -0.4650438129901886, 1.2559162378311157, -0.6715554594993591, 0.92079758644104, -0.4987412393093109, 2.241767406463623, -1.8118213415145874, -0.7292364239692688, -0.15351974964141846, 0.04949933663010597, 0.33362579345703125, 0.20676212012767792, 1.0058248043060303, 0.6671748757362366, 0.7025176286697388, -1.6367242336273193, -0.44642430543899536, 0.5624484419822693, -1.6275578737258911, 0.6430065631866455, 1.7421271800994873, 0.5064421892166138, 0.3927028477191925, -0.4027906060218811, 0.10142185539007187, -2.1473381519317627, -0.5211856365203857, 1.6567060947418213, 1.8046218156814575, -0.5868932008743286, -0.533297598361969, 0.9657196402549744, -0.4372674524784088, -2.271164655685425, -0.5368590950965881, 0.10919061303138733, 1.0188517570495605, -0.8728699088096619, 1.078896164894104, -0.27709582448005676, 0.8020364046096802, -0.23194444179534912, -0.8619430661201477, 0.20595695078372955, -0.46069562435150146, 1.3193503618240356, 0.5362458825111389, -1.7088760137557983, 0.16692687571048737, -1.4872400760650635, -0.273436039686203, -0.029613541439175606, -0.856357753276825, 0.04572592303156853, 0.8705607652664185, -0.4710141122341156, 0.33376961946487427, 1.788841962814331, -1.9061115980148315, -0.3745274543762207, 0.10113703459501266, 0.36539924144744873, -0.13930894434452057, 1.4309803247451782, -0.18514108657836914, -0.9150019884109497, -0.4672510027885437, 0.07383160293102264, -1.266390323638916, 0.3581775724887848, -1.236291527748108, 0.2258693128824234, 0.3668924570083618, -2.2026002407073975, -0.6296062469482422, 0.5408006906509399, 0.48395106196403503, -0.9115399122238159, 0.9833457469940186, -0.001425003632903099, -1.2921934127807617, -0.4440144896507263, -0.10477577149868011, 0.6589313745498657, -0.3987525999546051, -1.5026724338531494, 1.8151578903198242, 1.4035041332244873, 1.041152834892273, -1.9092472791671753, -1.2301946878433228, -0.17099282145500183, 0.2269759476184845, 0.36439916491508484, -0.10454782098531723, -1.8195480108261108, -0.2926601767539978, -0.8473390936851501, 1.6929974555969238, 0.36765047907829285, 0.7123465538024902, 1.946215271949768, -0.5196326375007629, 1.018608808517456, -0.17392483353614807, -1.1775736808776855, -1.1861146688461304, -1.159777283668518, 0.08554454892873764, 1.1186383962631226, 1.2704168558120728, 0.37948375940322876, -0.23221850395202637, -1.8868403434753418, -0.017127789556980133, -1.5212959051132202, -1.2460538148880005, -0.011341334320604801, 0.4620613753795624, 1.070072889328003, -0.08126316964626312, -2.2176952362060547, -0.08361570537090302, -0.12371766567230225, 1.5318559408187866, -1.327572226524353, 1.0601140260696411, -0.025323381647467613, 2.6671502590179443, 0.3568519055843353, 0.7698872089385986, -0.09178713709115982, 1.259093999862671, 0.21046920120716095, 0.18371941149234772, -1.747003197669983, -0.22444362938404083, -0.8933013677597046, 1.698796033859253, 0.5592896938323975, 1.4221038818359375, 0.0031861586030572653, 0.09271863102912903, 0.3642396628856659, 0.793137788772583, 0.2472517341375351, -0.957430899143219, 1.7460561990737915, -0.3778022229671478, -1.0281482934951782, -0.8177157044410706, -0.6827114820480347, 0.43357419967651367, 1.0982743501663208, 0.08823879808187485, -1.0357729196548462, 0.6960313320159912, 2.146876335144043, -0.695378839969635, 0.2908768057823181, -0.5039680600166321, 1.2060775756835938, 0.10426846146583557, 0.7544095516204834, 0.23322470486164093, 1.9753867387771606, -0.7782762050628662, -0.4755915403366089, -1.4185689687728882, 0.24171501398086548, 1.6410292387008667, -0.6410962343215942, -0.21031257510185242, 0.2222641110420227, 0.1851232498884201, 0.19957610964775085, 0.4114631116390228, 1.1666456460952759, -0.4797534942626953, 0.35631194710731506, 0.6863646507263184, 0.4479530155658722, 0.23786601424217224, -0.3471734821796417, -0.891596257686615, 0.2304663211107254, 0.01767769642174244, -0.6159993410110474, -1.2789239883422852, 0.20413844287395477, 0.3228779137134552, 0.12629076838493347, 0.600950300693512, -0.5262951850891113, 0.6482349634170532, 0.9704638719558716, -0.639819860458374, 0.6990261077880859, -0.23770153522491455, 0.769363284111023, 0.15152710676193237, -0.7698807120323181, 1.9080396890640259, 1.352946162223816, 1.4826542139053345, -0.9099506139755249, -1.185228943824768, -0.6585142612457275, 1.9646552801132202, -1.3300169706344604, -0.75830078125, -0.8233723640441895, -0.903008222579956, 0.26867571473121643, 0.46992063522338867, 0.8683251142501831, 1.0931607484817505, 0.051345232874155045, 0.4657188951969147, -0.9945974349975586, 0.5782656669616699, -0.5076309442520142, -0.08510556817054749, 0.028780950233340263, 0.6370168924331665, 0.023040270432829857, 0.5497539043426514, -1.6499696969985962, -0.7583009600639343, 0.9294679760932922, -0.5475252270698547, -0.2254086434841156, -1.923744797706604, -1.0958268642425537, 0.23816393315792084, -1.2125879526138306, -0.10938292741775513, -0.3572799861431122, 1.798755168914795, 0.6430290341377258, 2.3726840019226074, -1.9269253015518188, -1.145229697227478, -0.020932063460350037, -1.6808900833129883, -0.9795432090759277, -0.6024267673492432, 0.16014926135540009, 0.8719850778579712, -1.3644593954086304, -1.72291898727417, -0.27507805824279785, -0.9141498804092407, -0.00811493769288063, 1.7039798498153687, 1.9304198026657104, -0.885271430015564, 0.4194791316986084, -0.7169241309165955, -2.14280104637146, -0.980097770690918, -0.19900241494178772, 1.222503662109375, -0.514792263507843, 0.3138439655303955, -1.248562216758728, -0.9812290668487549, -0.4037553071975708, 1.6855647563934326, -0.8737847208976746, 1.2633754014968872, 1.1683956384658813, -0.45536744594573975, -0.3412526547908783, 1.6917691230773926, -0.8947595953941345, 0.8325216174125671, -0.291219562292099, 0.9913647770881653, 1.5872246026992798, -2.166520833969116, 1.1985571384429932, -0.08096082508563995, 0.7784759998321533, -0.9674908518791199, 0.3832622170448303, -0.06747215241193771, -0.059046193957328796, 1.0043003559112549, -0.23703911900520325, -0.4904862642288208, 0.1885511577129364, 0.14968618750572205, 0.7125041484832764, -0.38443654775619507, 0.6190862059593201, -1.1021960973739624, 0.09830920398235321, -0.15872105956077576, 0.6123224496841431, 0.6194924116134644, 0.4028644859790802, 0.13511213660240173, 0.88169264793396, 0.4489772319793701, 0.8469637632369995, -1.2656733989715576, 0.015231398865580559, 1.009141206741333, -0.5642873048782349, 0.0041691311635077, 0.08698797971010208, 0.6760250329971313, 0.024820195510983467, -1.1128346920013428, -0.9545204043388367, -1.0219229459762573, 0.6947751045227051, -0.31563088297843933, -0.9025273323059082, -1.3479083776474, -1.1311546564102173, -1.1835178136825562, 0.263731449842453, -0.19636587798595428, 0.5832900404930115, -0.07265068590641022, 0.2765989303588867, 0.3729473650455475, -0.8500342965126038, -0.56980299949646, 0.7025467157363892, -0.3030160069465637, 0.5526208281517029, 0.3490649461746216, -1.8812857866287231, 0.9938083291053772, 0.3670872747898102, 0.15148648619651794, 0.44955185055732727, -1.1160640716552734, -1.5062350034713745, -2.358440399169922, 1.655350685119629, 0.35192131996154785, -0.1779651939868927, -0.6315242648124695, -0.0009444375755265355, -1.3386884927749634, 0.8560194969177246, -0.9052817821502686, -0.1415284276008606, 1.1022047996520996, -0.20901556313037872, 1.043506145477295, 0.43459656834602356, -0.9441616535186768, -0.33355075120925903, -0.6372923851013184, 1.9127188920974731, 1.1424120664596558, 0.08439330756664276, -0.3164346516132355, 1.0898339748382568, 0.6800527572631836, 1.0465140342712402, -0.4239630103111267, 1.351771593093872, 0.8490312695503235, -0.9009369015693665, 0.38200902938842773, 0.06050239875912666, 1.6096742153167725, -0.7196207046508789, 0.7112134099006653, -0.2177162915468216, -2.2535781860351562, -0.3261989653110504, -0.06532304733991623, -0.1084180474281311, -0.5960172414779663, 1.5189942121505737, -0.06988159567117691, -0.3229502737522125, 0.6044110059738159, -0.500759482383728, 0.8741104006767273, -1.288952350616455, -0.36207765340805054, 1.072638988494873, 0.5819505453109741, 0.284665048122406, 0.5200006365776062, 0.27197930216789246, 1.134864330291748, -0.157740980386734, 0.5424132347106934, 0.3712180554866791, -0.6468223333358765, 0.5692635774612427, -0.14852958917617798, 0.3180184066295624, 1.789618730545044, -0.1375243067741394, 0.4413290321826935, -1.0851837396621704, -0.14981962740421295, -0.5499888062477112, 1.956817865371704, -0.4642484486103058, -0.4551914930343628, -1.0364748239517212, -0.6813780069351196, 0.6112709641456604, -0.33949652314186096, 1.8598380088806152, -1.079519271850586, -0.8324075937271118, 0.6768168807029724, -0.4398081600666046, 0.04763086885213852, -0.6405632495880127, -1.4585473537445068, 1.3403867483139038, -1.5185840129852295, -0.4743490219116211, 0.9377159476280212, -0.8820680379867554, 0.6180513501167297, -0.11894094944000244, 0.4308560788631439, -0.17504648864269257, -0.33703118562698364, 0.015809355303645134, 2.8482766151428223, -0.20674721896648407, -0.5879995822906494, 1.0803472995758057, -0.17724503576755524, 0.14612998068332672, 0.7179349064826965, -0.12034408003091812, 0.5278449654579163, -0.6195865869522095, -0.6216335296630859, -1.3895584344863892, -2.4171223640441895, 0.530381441116333, -1.6517475843429565, -0.7311391234397888, 0.5304789543151855, -1.5011858940124512, 0.2613300383090973, -0.2895341217517853, 2.124282121658325, 0.5028020739555359, 0.16594184935092926, 0.35699981451034546, -0.9940590858459473, 1.5871611833572388, 1.2505900859832764, 0.779338002204895, 0.601679801940918, 0.5666240453720093, 0.26615506410598755, -2.63264536857605, -2.638550043106079, -0.5214988589286804, 0.9966432452201843, -0.8631345629692078, -1.8941599130630493, 1.3884981870651245, 0.7335399389266968, -0.8112612962722778, 0.44031521677970886, -0.29832449555397034, 0.2677818238735199, 0.3644002079963684, -0.45274603366851807, 0.7296082377433777, 0.33302977681159973, -0.5629299879074097, -0.7821050882339478, 0.4665013551712036, -1.1143425703048706, 0.12894998490810394, -1.452301263809204, 0.9373947978019714, -0.7775041460990906, 0.05415600538253784, -0.5648153424263, -0.42788735032081604, 0.4130082428455353, 0.08424214273691177, -1.1228059530258179, -0.9847894310951233, -0.6806371212005615, 0.6268003582954407, 0.5905687808990479, 0.695343554019928, 0.8792303800582886, -0.6402199864387512, -0.24401246011257172, -0.8037450909614563, 0.1196865439414978, -0.7485033869743347, -0.04931812733411789, 0.01862148754298687, -0.2859671413898468, -0.9332939386367798, -0.5359026193618774, -0.9788044691085815, -1.398152232170105, -0.1281561255455017, -0.22881990671157837, -1.987694501876831, -0.029154937714338303, 2.4579720497131348, 0.35902997851371765, 0.18635903298854828, -0.44746893644332886, 0.7914049029350281, -0.3114897310733795, 0.20004478096961975, 0.4228821396827698, 0.7883379459381104, 0.4162251055240631, -0.25358277559280396, -0.2576914429664612, -1.4506471157073975, -0.8583999276161194, -0.14219577610492706, -0.7252718210220337, -0.5623708963394165, -2.811981201171875, -0.1806917041540146, -0.03861372172832489, -0.736581563949585, 1.3194583654403687, 1.3296154737472534, 0.1770791858434677, -0.6962448954582214, -1.009661078453064, -2.008700370788574, 2.9125354290008545, 0.03405033051967621, 0.709854245185852, -1.928911566734314, 1.4482390880584717, 1.2038129568099976, -0.08726362884044647, 0.04041944444179535, 0.9501346945762634, 1.4219011068344116, -1.6565005779266357, 2.761087417602539, -0.1761157363653183, 1.7384315729141235, 0.410762757062912, -0.4767882525920868, -0.16996999084949493, 0.9170942902565002, -2.2112178802490234, -0.20732595026493073, 1.2444016933441162, -0.6968103647232056, -0.347159743309021, -0.027943968772888184, -1.762538194656372, -1.263563871383667, 0.08898182958364487, 1.4920220375061035, -1.1111193895339966, -0.9704895615577698, -1.0010277032852173, 0.6066544055938721, 0.33683696389198303, -0.13109400868415833, -0.07279585301876068, 0.0026908947620540857, 1.9378209114074707, 1.2095799446105957, -0.9147128462791443, -0.009451832622289658, -0.7971434593200684, -1.1203958988189697, -0.47328126430511475, -0.6815357804298401, -2.255026340484619, -0.21007569134235382, 0.46820393204689026, 0.7887874245643616, 0.5123413801193237, -0.6118924021720886, -0.7658868432044983, 2.140507936477661, 0.3936231732368469, 0.033654261380434036, -0.8921194672584534, 2.3434762954711914, -0.6108322143554688, -0.6057718396186829, 0.8054404854774475, -2.5926434993743896, 1.8230226039886475, -0.18035824596881866, 0.19590488076210022, 0.95087069272995, 0.5919889807701111, -1.0328086614608765, 0.17750367522239685, -0.8381165266036987, -0.021919341757893562, 1.281882643699646, -0.20413516461849213, 0.12996558845043182, 2.194648265838623, 1.912145733833313, 2.5259528160095215, -0.08141310513019562, 0.3887469172477722, 0.7530346512794495, 1.8498343229293823, -1.1102149486541748, -0.013123055920004845, 1.077938437461853, -0.9657742977142334, 1.1570403575897217, -0.15818250179290771, 0.17020803689956665, 0.36949172616004944, 1.94888174533844, -0.07009831815958023, -1.0566515922546387, -2.3593263626098633, -0.28415271639823914, -0.8954117298126221, 0.14882177114486694, 0.7083882093429565, -0.31593915820121765, -0.14150302112102509, -0.7554839849472046, -0.9355820417404175, 1.0103179216384888, -0.45941609144210815, 1.1217023134231567, -0.24076147377490997, -0.8284017443656921, -0.2767406105995178, 0.4407317340373993, 0.7891645431518555, -1.8193880319595337, -1.1984076499938965, 0.6576281189918518, 0.6559277772903442, -0.1267486959695816, 0.8934386372566223, -0.4964525103569031, -0.8349632024765015, -0.8506271243095398, 1.2087712287902832, 0.018267931416630745, -2.6213347911834717, 0.15210074186325073, -1.137427568435669, 1.0136959552764893, 0.6603850722312927, 0.08491736650466919, -1.203634262084961, -0.6076046824455261, 0.8945364356040955, -0.16860714554786682, -0.7388626933097839, -2.3776516914367676, -1.1133825778961182, 1.1165835857391357, 1.2014015913009644, 1.1022326946258545, 0.36664021015167236, -0.08341681957244873, -1.684697151184082, 0.008012608624994755, 0.39483317732810974, 2.3964338302612305, -0.8034241199493408, 0.23162798583507538, -1.9004443883895874, 0.5137620568275452, 0.13966476917266846, 3.021608829498291, -0.3153054714202881, -1.411253809928894, -0.11852093786001205, -0.5951050519943237, -0.36844614148139954, 0.10460153222084045, 2.756256103515625, 0.39173731207847595, -0.15937311947345734, 0.641686201095581, 0.5025123953819275, -0.47385987639427185, 0.9347146153450012, 1.0700691938400269, 1.6994726657867432, -0.3154020607471466, -1.0152723789215088, 0.4946848452091217, -1.2454378604888916, -0.00904060248285532, -0.5682109594345093, -1.1716899871826172, -1.9054652452468872, -0.07998691499233246, -0.1764497458934784, -1.1676008701324463, 0.6415720582008362, 0.9105142951011658, 0.5276011824607849, 0.7679852247238159, 1.168854832649231, 1.6014125347137451, -0.01795954816043377, 0.9028361439704895, -0.4215417504310608, 0.39150527119636536, 0.10598764568567276, -1.8428822755813599, 0.6651989817619324, -1.3243995904922485, -0.3495842516422272, -0.378814697265625, -1.625175952911377, -1.161210536956787, -1.0512051582336426, 0.0201487448066473, -1.2515573501586914, -0.38328665494918823, -0.5303913354873657, 0.8157851099967957, -0.16374172270298004, -0.24631084501743317, -1.6687769889831543, -0.8413463830947876, 0.6076480746269226, -0.21419726312160492, 0.31569862365722656, 1.5904263257980347, -1.105205774307251, -0.589925229549408, -1.5415184497833252, 0.21485978364944458, -0.34807777404785156, -0.00720023550093174, 0.335222065448761, -1.4140583276748657, 0.6386466026306152, 0.500047504901886, 0.6465128660202026, 1.3079257011413574, -0.07724599540233612, -0.18257948756217957, -0.3044450879096985, 0.3251415491104126, 1.4639780521392822, 0.5698438882827759, -2.8460607528686523, -0.19586166739463806, -1.0503244400024414, 0.2908268868923187, 1.3751343488693237, -2.511143684387207, -0.09304551035165787, 0.7612640857696533, -0.20797553658485413, 0.492837131023407, 2.323976516723633, -0.031516190618276596, -0.652353823184967, 2.9740066528320312, -0.4239003360271454, 0.6393640637397766, -0.983996570110321, 0.4218790829181671, -1.5929198265075684, 0.4104182720184326, 1.4042521715164185, -0.6187072992324829, 0.2109520137310028, -1.544141411781311, 1.2092833518981934, -0.929631233215332, -1.5270155668258667, -0.6307512521743774, -0.0090468879789114, -0.9157630801200867, 1.036275863647461, 0.4488826096057892, -2.169754981994629, 0.45801976323127747, -1.793839693069458, -2.8634426593780518, -0.16157963871955872, 0.14124125242233276, -0.24937833845615387, -0.19041267037391663, -0.06956256926059723, 0.21427205204963684, -0.291299968957901, 0.10388512164354324, 1.9961020946502686, 0.7753354907035828, -1.2866082191467285, -0.4171789586544037, 1.4193485975265503, -0.8544335961341858, -0.29537540674209595, -0.19459611177444458, 0.9641969799995422, -0.021113181486725807, -1.4143136739730835, 0.109763503074646, -0.9278010129928589, -1.1849040985107422, 0.7310163378715515, 0.5321818590164185, -0.46428272128105164, 0.9737992286682129, 0.1339077204465866, -0.47530248761177063, -0.4336058497428894, -0.8635377287864685, 0.9331193566322327, -1.3460273742675781, 0.9843671917915344, 0.011510130949318409, 0.3890647888183594, -0.7884865999221802, 0.30133381485939026, 0.041109368205070496, -0.4586348831653595, -1.173987627029419, -0.4171799421310425, 0.05852973088622093, -1.1796212196350098, -0.3351207673549652, -1.2807050943374634, -0.11030028015375137, -0.3745615780353546, 1.0657051801681519, -0.753398060798645, 0.8768109679222107, 0.6977279782295227, -0.8076303601264954, 0.5483157634735107, 0.6926255226135254, 1.8456530570983887, -0.8379318714141846, -1.022286057472229, 0.011809133924543858, 0.7727862596511841, -1.4608955383300781, -1.1490952968597412, 2.0462934970855713, 0.19765642285346985, -0.5448010563850403, 0.08668969571590424, 0.7635260820388794, -1.320897102355957, -0.2564253509044647, -1.6624488830566406, -0.6373555064201355, 0.3455386459827423, 0.5446860790252686, -2.1593270301818848, -1.8336081504821777, -1.1600878238677979, -1.7438931465148926, 0.49778124690055847, -0.21995633840560913, 1.0564136505126953, -1.5408744812011719, 0.36626964807510376, 1.1748498678207397, 0.3157300353050232, 0.5050640106201172, 1.0902738571166992, -0.3707434833049774, 0.3023267388343811, -1.224068284034729, -2.14302921295166, 1.0721122026443481, -1.3695875406265259, -0.5630028247833252, 0.36349815130233765, -1.118181586265564, -0.2713063955307007, 0.5162196159362793, -0.2393958866596222, -0.34522750973701477, 1.5013474225997925, -0.44214218854904175, 0.4047297239303589, 0.020399458706378937, -1.4547228813171387, -1.4885793924331665, -0.34763866662979126, -1.2204753160476685, 1.163216471672058, 1.0723387002944946, -1.09635329246521, -2.480863571166992, 0.8252143859863281, 0.036047399044036865, 0.9013445377349854, 0.7296936511993408, -0.8221355080604553, 0.3627203106880188, -1.296182632446289, 2.0928685665130615, -2.1290292739868164, -0.16295106709003448, -0.009031313471496105, 0.223809614777565, 0.39574652910232544, 0.5548797845840454, -0.36936596035957336, -1.169821858406067, -0.04369831457734108, -0.4895671308040619, 1.634895920753479, 0.18578647077083588, 0.23500403761863708, -1.163192629814148, 0.794478714466095, -0.37389546632766724, -0.120643749833107, 0.7320036888122559, 0.7793755531311035, 0.015312984585762024, 0.289379358291626, -0.6284270286560059, 0.5483136177062988, -1.1666603088378906, -0.5293357372283936, -0.03986053541302681, 0.7495414614677429, 1.4077988862991333, -0.6358193755149841, -0.33782297372817993, -0.2629289925098419, 0.5921463370323181, -0.0664362758398056, 0.5895664691925049, 0.9633809924125671, 0.2969822287559509, 0.9328383207321167, -0.10775784403085709, -0.1551610231399536, 0.7996888756752014, 0.5390059351921082, 0.8719013929367065, -1.6063588857650757, 0.22133195400238037, 1.4316881895065308, -1.061457872390747, 0.749248743057251, -0.7414993643760681, -1.4548283815383911, 0.9955520629882812, 0.6417886018753052, -0.25413650274276733, 0.0006210154970176518, 0.10398479551076889, -0.23388995230197906, 0.39027804136276245, 1.0804815292358398, 0.45671430230140686, -0.6527895331382751, -2.831446647644043, 0.30899691581726074, -0.4859631061553955, -0.2366228997707367, 0.5472438335418701, -0.49568602442741394, -0.18778753280639648, 0.04127994924783707, -0.8327833414077759, 1.1807405948638916, 0.325443297624588, 0.2603563368320465, 0.4914170801639557, -0.13101592659950256, -1.235203742980957, 1.4542570114135742, 0.43914273381233215, 0.6525079011917114, 1.4532843828201294, -0.38851451873779297, 0.04644493758678436, -0.42527127265930176, 0.5513038039207458, 0.7237297296524048, 0.6294272541999817, -0.22182314097881317, -1.3094209432601929, 0.3593394458293915, 0.7119985222816467, 0.9373540282249451, 0.7625026106834412, 0.6130746006965637, 0.1648845374584198, -0.12286747992038727, 0.20053990185260773, 1.1892796754837036, -1.0331275463104248, 0.7364597320556641, 0.217549666762352, 0.04063405841588974, -1.0170869827270508, 0.690739631652832, 0.09380439668893814, 0.9148287773132324, 0.4520924687385559, 1.0617053508758545, -1.7394038438796997, -0.9537864327430725, -1.4335393905639648, 0.776508092880249, -0.0870562419295311, 0.4960384964942932, -0.7747814059257507, 0.4390399158000946, 0.9033698439598083, -1.2211976051330566, -1.2257905006408691, -0.5704699754714966, 3.008784770965576, -0.8353262543678284, -0.24300450086593628, -0.42030951380729675, -0.29167357087135315, -1.5508934259414673, -0.9403021335601807, 1.4928728342056274, 0.5643149018287659, 2.4130196571350098, -0.12360353767871857, -1.867287516593933, 1.2792375087738037, -0.5472091436386108, 0.21794182062149048, -0.4841896593570709, -2.1745243072509766, 1.4856412410736084, 1.0844656229019165, -0.11241858452558517, 0.5762654542922974, 0.10492414981126785, 0.6428159475326538, -0.13786157965660095, 1.3545737266540527, 1.2046722173690796, -0.26595166325569153, 0.634956419467926, -0.6749216914176941, -2.290045738220215, 0.08513065427541733, -0.34113946557044983, 0.4384010136127472, -0.09936807304620743, -0.7836265563964844, -0.030732115730643272, -0.03196355327963829, 1.6233959197998047, 0.0500316396355629, 0.3468681573867798, -0.372214138507843, -2.544606924057007, 0.8430852293968201, 0.5866175293922424, -0.2927025258541107, 0.039361730217933655, -1.092071533203125, 1.5052528381347656, -0.28420472145080566, -1.5470318794250488, -1.870479702949524, -0.3997061252593994, -2.1597225666046143, 0.7944006323814392, 0.9202451109886169, -1.882743239402771, -0.3725866377353668, -0.7768163681030273, -2.2250638008117676, -0.09635990113019943, -0.28875765204429626, -0.45602157711982727, -1.3569167852401733, 0.24580436944961548, 0.010373338125646114, 0.8141520619392395, 1.4856271743774414, 0.25346800684928894, 0.0003220686048734933, -0.7207912802696228, 0.6968822479248047, 0.8627222776412964, 0.2838602364063263, 0.3473505675792694, 0.3824796676635742, -0.7185060381889343, 0.36988312005996704, 0.7970702052116394, 1.9994183778762817, -0.1899261176586151, -1.3646278381347656, -0.6035007834434509, -1.38776433467865, -0.32005155086517334, 0.6273326873779297, 0.6311349272727966], [0.18906466662883759, -1.695784091949463, -0.238302543759346, 0.13226568698883057, -0.22296050190925598, -1.9394969940185547, 1.1274166107177734, 0.3501829504966736, -0.9948587417602539, 0.9437162280082703, -0.5490884184837341, -0.45605337619781494, -0.7020871639251709, -0.7286175489425659, -1.29897141456604, 0.7280660271644592, 1.9855983257293701, 0.4495631754398346, -0.6898014545440674, -0.25881606340408325, 1.6584498882293701, 0.5469526052474976, 0.8866094350814819, 1.9348504543304443, -0.4835060238838196, 0.0386962965130806, -0.4675544202327728, -0.5071143507957458, -0.1990661472082138, 0.41257649660110474, 1.090190052986145, 0.05725196748971939, -2.4394683837890625, 0.16413313150405884, 0.6425231099128723, -1.3217182159423828, 0.41594570875167847, 0.26663804054260254, 0.6878982186317444, 0.35879436135292053, 0.3917767405509949, 0.030587246641516685, -0.39852601289749146, -2.2006378173828125, 0.3364594578742981, 0.5076438188552856, 0.30279940366744995, -0.5470012426376343, 0.5018637776374817, -0.39448171854019165, 0.5894883275032043, -0.6731241345405579, -0.3070701062679291, -0.4950874149799347, -0.14763522148132324, -0.7842221856117249, 1.4585107564926147, 0.4583316147327423, -0.23108884692192078, -0.14521610736846924, -0.3418585956096649, -1.0800930261611938, -1.7386584281921387, 0.18547281622886658, 0.7224788665771484, 1.2618657350540161, -0.37445780634880066, 0.5205332636833191, 0.5739864110946655, 1.2277238368988037, 0.9569652676582336, -0.2676835060119629, 0.2873901128768921, 0.40937912464141846, -0.5161663293838501, -0.8105327486991882, -0.55150306224823, 0.759739100933075, 0.11361156404018402, -0.8041409254074097, -0.8358434438705444, 1.487052321434021, 0.44125500321388245, -0.33778640627861023, -1.9296715259552002, 1.0802762508392334, 1.7376428842544556, 0.09428732097148895, -2.092057704925537, -0.2702818512916565, 0.5279473662376404, 0.4662342965602875, -0.3324020802974701, -0.7100593447685242, 0.5535268187522888, 0.6569796800613403, -0.5257842540740967, 0.9201272130012512, -1.2387179136276245, 0.5302673578262329, 0.9239591956138611, 1.4219059944152832, -0.9329825639724731, 0.43737921118736267, 2.3074779510498047, -0.1767948865890503, -0.11991357803344727, 1.7351607084274292, 0.45569586753845215, -0.0006965105421841145, 0.9016872644424438, 0.7384002804756165, -0.0038594717625528574, -1.0089023113250732, -0.1668856143951416, -0.7123710513114929, 1.562368392944336, -0.4790680706501007, 0.69387286901474, 0.48939791321754456, -0.6072818636894226, 2.3301279544830322, -1.014045000076294, -0.9494695067405701, -2.2305357456207275, 0.05757306143641472, 0.8324621319770813, -1.0221108198165894, 0.6975330114364624, -0.4923878610134125, -0.959575355052948, -1.3345859050750732, 2.543576240539551, 0.39641088247299194, 0.717261016368866, -1.1482852697372437, 0.07203415036201477, 0.37525805830955505, 0.5192522406578064, 0.8888890743255615, 0.2941431403160095, 0.4540256857872009, -1.1684595346450806, 1.1575955152511597, -0.23377326130867004, -0.741804301738739, 0.6680045127868652, 0.18051794171333313, -2.566983461380005, -1.4717936515808105, 1.3211756944656372, -0.510631799697876, -1.7604550123214722, -0.9718945026397705, -0.4058959186077118, -0.6592003107070923, -0.15773971378803253, 1.3494917154312134, -0.4982892870903015, 2.2011289596557617, 0.3301888108253479, 0.8075870275497437, 0.9095783233642578, 3.0905370712280273, -1.0649555921554565, 0.34417837858200073, 0.36767351627349854, 1.288510799407959, 1.483942985534668, 1.0992220640182495, -1.4460948705673218, -0.8443228006362915, 1.111756443977356, 0.9131349921226501, 0.9228905439376831, -1.2225074768066406, -0.273104727268219, -1.2284824848175049, 0.8867996335029602, 0.43741199374198914, -1.0561330318450928, 2.087787628173828, 0.6868743896484375, -0.7926718592643738, 1.4551124572753906, 1.5099557638168335, -0.28477975726127625, -0.9516122341156006, 1.1197041273117065, -1.0836000442504883, -0.1669842004776001, -1.4190207719802856, -0.9779295921325684, -1.5951521396636963, -0.9928027987480164, -0.2966829240322113, -1.2420008182525635, -0.3676171600818634, 0.06276658177375793, -1.2293351888656616, 0.8099971413612366, -0.525964617729187, 0.8787316679954529, -0.4510935842990875, -0.7626051902770996, -0.5617415308952332, 0.9622421860694885, -0.056207410991191864, -0.9819685220718384, 0.3341510593891144, 1.3199267387390137, 0.05518108233809471, -0.31308412551879883, 0.3222358524799347, 0.2437126487493515, -0.34176626801490784, 1.8736532926559448, 0.154762864112854, 0.9170194268226624, 0.14781907200813293, -0.2534322738647461, -2.4499824047088623, 0.02818213775753975, -0.2149701714515686, -0.5519530773162842, 0.8236075639724731, -0.02407793328166008, -1.259574055671692, 0.9643155932426453, -2.0016705989837646, -0.7835096120834351, -1.4622845649719238, 0.20068296790122986, -0.07600244879722595, -1.3078796863555908, -0.5742235779762268, 0.6360781192779541, -1.6455082893371582, -0.34838593006134033, -0.8365997076034546, -1.8422770500183105, -0.5341362953186035, 0.47387945652008057, -0.14258618652820587, -1.3213914632797241, -2.1304452419281006, 0.09602611511945724, -0.1537790298461914, 0.23161360621452332, -0.03454361483454704, 1.12373685836792, 0.5492866039276123, 0.5361461639404297, 1.4296233654022217, 1.0962285995483398, 0.45173725485801697, 2.1756176948547363, 0.4627268314361572, -1.0205833911895752, 0.1786973625421524, -0.6513407230377197, -1.266762137413025, 0.20114877820014954, 1.3650834560394287, -1.5403521060943604, 1.4201704263687134, -0.12056515365839005, -0.28393542766571045, 0.0823911800980568, 0.409148633480072, 0.2425239086151123, 2.2739689350128174, -1.1562937498092651, 1.2871911525726318, 0.3061583340167999, -0.3986823260784149, -0.30953747034072876, -0.22617857158184052, 0.19915427267551422, 0.5507158637046814, 1.0389156341552734, 1.6439738273620605, -0.7876169085502625, 0.15190552175045013, 1.2771356105804443, 0.5373921394348145, -0.9785148501396179, -0.0017445581033825874, 0.23093155026435852, 0.3644678592681885, -1.5569109916687012, -0.6021026372909546, 0.631466805934906, -0.24430331587791443, -1.6185697317123413, 1.0642062425613403, 1.2017159461975098, -1.0407859086990356, 0.6031026840209961, -1.0690221786499023, 0.7208902835845947, -0.38689160346984863, 0.9429667592048645, 0.3666425943374634, -0.3800908625125885, 0.6269286870956421, -1.0195941925048828, 0.4058118462562561, -1.5697882175445557, 0.0015375185757875443, 0.14920783042907715, -1.0349419116973877, -0.04487033933401108, 0.6434809565544128, -2.4999570846557617, 0.677003800868988, 0.265848845243454, -0.5996853113174438, -1.6005487442016602, -2.3663463592529297, -0.059934113174676895, 0.7335631251335144, 0.5632102489471436, 2.179816246032715, 0.31255826354026794, -0.1954733282327652, -0.5787429213523865, -0.2821883261203766, -0.34345167875289917, -0.2403782606124878, 0.5471432209014893, 0.9729377627372742, 0.3884202837944031, 0.03760450705885887, -1.4747295379638672, 0.7485731244087219, 0.5722677707672119, -0.2881016135215759, 0.6343083381652832, -0.7024849057197571, -0.5596035718917847, 0.2844959497451782, 1.9019732475280762, 0.13166247308254242, 1.2550538778305054, 1.4367579221725464, 1.3187702894210815, 1.305092215538025, 0.2850261330604553, -0.33370912075042725, -2.320863723754883, 0.38003140687942505, -0.05517003312706947, -1.302726149559021, -1.172308325767517, 0.111366868019104, 1.044037938117981, 0.21177029609680176, 0.21619686484336853, -0.23652632534503937, -0.008870182558894157, 1.0188004970550537, 1.297404170036316, -0.018862372264266014, -1.1051921844482422, 0.4169082045555115, 1.6548436880111694, -0.1991974413394928, 1.1770151853561401, 0.9078202247619629, -2.7293307781219482, 0.5852043032646179, -2.671992540359497, 0.9033976793289185, 1.426221251487732, -0.027593769133090973, -0.0918271392583847, -0.43212372064590454, 1.9965660572052002, 0.443410187959671, -0.1928330361843109, -0.05780297890305519, -0.7733646035194397, -0.17574764788150787, 1.1585015058517456, 0.5599997639656067, 0.8081073760986328, 1.1470012664794922, 0.1922348290681839, -1.6387085914611816, -0.012152730487287045, 0.38095688819885254, 1.1058849096298218, -0.07777288556098938, 0.5385866761207581, 1.7591463327407837, -0.07611145079135895, -0.3118213713169098, 0.36160534620285034, 1.8156243562698364, 1.1356639862060547, 0.608169674873352, -0.5087509751319885, -1.1113193035125732, -0.4441082179546356, 1.3663578033447266, -1.5733333826065063, -0.2865455746650696, 0.8097206354141235, -0.9723192453384399, 0.9131670594215393, 0.24272865056991577, 1.1205520629882812, 0.1934967190027237, -0.19868850708007812, 0.9171271920204163, -0.531065821647644, 0.5385504961013794, -0.9952021241188049, 1.539199948310852, 0.88228440284729, -0.12281034886837006, 0.9050129055976868, 1.2403110265731812, 0.43423935770988464, 0.1971718817949295, 0.00036142015596851707, -0.43383076786994934, -1.488384485244751, 0.6488706469535828, -1.9746193885803223, -2.4470484256744385, 0.8717919588088989, 0.2111101597547531, 0.682684063911438, 0.8930428624153137, 0.7428656816482544, 0.6685258746147156, 1.1701891422271729, 0.5916616320610046, 0.4825485944747925, 0.0038714739494025707, 0.5046852827072144, 1.728275179862976, 0.2610824704170227, -0.5867379903793335, -0.6585408449172974, -0.7162305116653442, -0.17388971149921417, -1.0037983655929565, 0.6527949571609497, -0.542175829410553, -1.1497865915298462, -0.8873955607414246, 0.5171854496002197, 2.1403815746307373, -0.4696541130542755, 0.4154439866542816, 1.3450067043304443, 0.8683132529258728, -0.19992497563362122, -1.1034307479858398, -1.4871761798858643, -0.24214385449886322, -0.8360080122947693, 1.1588895320892334, 1.146791934967041, -0.8149833083152771, 1.3403327465057373, 1.1463277339935303, 0.6365511417388916, 0.9562512040138245, -0.15135276317596436, -0.6172360777854919, 1.0034393072128296, 0.5162315964698792, 0.3328186869621277, 1.5544158220291138, -0.09668939560651779, 0.21471546590328217, 0.39753058552742004, 0.1001194640994072, -1.943913221359253, 0.5338232517242432, -0.08524986356496811, 0.11774526536464691, 0.1989464908838272, 0.5906761288642883, 0.5258367657661438, 0.7116095423698425, -0.2735048234462738, 0.22151459753513336, 0.8060279488563538, -1.319542407989502, 0.17849883437156677, -1.0967563390731812, 0.9877346754074097, 0.7614672780036926, -0.14779698848724365, 1.5538924932479858, -1.202980875968933, 0.021957462653517723, -1.0591585636138916, 0.8581293225288391, -0.08022507280111313, 0.7608322501182556, 0.0658842995762825, -0.1984647512435913, -1.166196584701538, 0.7276686429977417, -0.7934562563896179, -0.9911741614341736, -0.733374834060669, 0.7983412742614746, 1.438063383102417, 0.6120776534080505, 0.769134521484375, 1.5812698602676392, 0.32875683903694153, -0.6016169786453247, 0.5200664401054382, 1.2855756282806396, 0.2790091335773468, 2.15771484375, -0.7839892506599426, -0.2835506796836853, -0.8637877702713013, 0.10491256415843964, -0.1543884426355362, -0.41731420159339905, -0.44840574264526367, 0.04914535582065582, 0.6500530242919922, 1.3540292978286743, -0.5581386685371399, 0.035793889313936234, 0.7217614054679871, -0.6051105260848999, -0.7499956488609314, -2.546915054321289, 0.3772425055503845, 0.7456338405609131, 0.281658798456192, -1.1780381202697754, 0.4324873387813568, 0.28891873359680176, -0.6904506087303162, 1.0138943195343018, 0.18214601278305054, 0.651258111000061, -2.093937635421753, 0.030770281329751015, -0.17847713828086853, -0.2672005593776703, -0.3233930766582489, -0.6053787469863892, -0.3442949950695038, 2.027409315109253, -1.0892404317855835, 0.5066009163856506, -1.0055253505706787, 1.6120152473449707, -0.588987410068512, -0.404759019613266, -0.03068680316209793, -1.1639108657836914, 1.1856544017791748, 0.2865322530269623, 1.0993951559066772, 1.5405515432357788, 0.9194950461387634, 0.050869088619947433, 2.0123496055603027, -0.7230669856071472, -0.29804009199142456, 0.5410785675048828, 0.5808771848678589, 0.3846074342727661, -0.868696928024292, 0.06740410625934601, 1.2313517332077026, 0.8531168699264526, 0.4396909177303314, 0.10529863089323044, -0.31843438744544983, 0.6375857591629028, 1.6476671695709229, 1.2894887924194336, -0.6080324053764343, -0.16736756265163422, -0.06303451210260391, -1.3588862419128418, 0.38816583156585693, 0.6020118594169617, -0.9751248359680176, -1.3586227893829346, 1.0056878328323364, 0.2667406499385834, -0.2418149709701538, 1.9332412481307983, -1.268805742263794, 1.0428493022918701, 0.7391679286956787, 0.18818046152591705, -0.015328599140048027, -0.6391578316688538, 1.2932956218719482, 2.326563835144043, 1.0635360479354858, 0.022759882733225822, -0.32243841886520386, -1.0202274322509766, 0.4398305416107178, -1.0905219316482544, 0.9628370404243469, 0.5326675772666931, 0.302954763174057, 1.3073674440383911, -3.394202709197998, 2.5207250118255615, 0.29956430196762085, -1.1753606796264648, 0.21522553265094757, 0.34950026869773865, 0.9278119206428528, -2.0446197986602783, -0.5584334135055542, 2.742349624633789, -0.534080445766449, 0.2925652861595154, -0.51056307554245, 0.5871586799621582, -0.5823162198066711, -1.4247620105743408, 2.099827527999878, 0.23575849831104279, 0.20480726659297943, -0.37787172198295593, 0.3232809603214264, 0.5798158049583435, 1.4650219678878784, -0.9052669405937195, 1.3054064512252808, 1.006973385810852, -0.021978115662932396, -1.0648140907287598, -0.30682459473609924, -0.0950460210442543, 1.1611686944961548, -0.14057068526744843, -0.9369905591011047, -0.083205945789814, 2.0371408462524414, -0.14993704855442047, 0.05704266577959061, 1.052337646484375, 0.3698345422744751, -0.007714959792792797, 0.4793162941932678, 0.4051300585269928, -1.4641274213790894, -1.2521153688430786, -1.6600048542022705, 1.2652033567428589, 1.2623659372329712, -0.9227244257926941, -0.36405304074287415, -0.9239353537559509, -0.053916361182928085, 0.14731110632419586, 2.015169143676758, -0.05698753520846367, 1.8805795907974243, -1.6697794198989868, 1.1091121435165405, -0.812684178352356, -0.5053858160972595, 0.17367807030677795, -2.315213441848755, -0.5902934670448303, 0.7967305183410645, 1.935174822807312, -0.9060985445976257, -0.89872807264328, 0.2564641237258911, 0.40984323620796204, 0.29172518849372864, -0.03406071290373802, 0.5949801802635193, -0.44074228405952454, -1.5856518745422363, 0.47202619910240173, 0.5207944512367249, 1.3510048389434814, -2.100154399871826, -0.6395897269248962, 0.7420458197593689, -0.3317152261734009, -0.3485700488090515, -0.5228983759880066, 2.063624858856201, 0.06011960655450821, -1.2197511196136475, 0.9943854808807373, 0.9476432800292969, -1.627307415008545, 1.12163245677948, -0.2019922435283661, 1.737622618675232, 0.19366125762462616, -1.190651774406433, -0.2166559100151062, -2.6225059032440186, -0.4670325219631195, 0.14113688468933105, -0.2570996880531311, -1.0347968339920044, 0.6148632168769836, -0.37971973419189453, -0.5748671293258667, -1.8876285552978516, 1.258285641670227, -0.21683575212955475, -2.213404893875122, -0.8439642786979675, 0.9263184070587158, 0.7234667539596558, 0.6659275889396667, 1.8478150367736816, 0.6525042653083801, -0.3566541075706482, 0.7860165238380432, 0.20089761912822723, -0.9426977634429932, -0.9272936582565308, 0.7146748304367065, -0.6360394954681396, 1.3740203380584717, 0.17942190170288086, 1.4656604528427124, -0.32943591475486755, 2.6883373260498047, -0.5019999146461487, 1.7937180995941162, -0.5589315295219421, -1.3702690601348877, 1.108992338180542, 2.2730464935302734, 0.8596386313438416, 0.5448567271232605, 0.24280773103237152, -1.066595196723938, -1.041121244430542, 0.25569725036621094, 0.008180678822100163, 0.211593359708786, 0.27637529373168945, 0.7764255404472351, 0.3200867772102356, -0.4931078553199768, -1.4634042978286743, -0.3425239622592926, -0.4282916486263275, 1.474362850189209, 0.365707129240036, -0.5865411758422852, 0.2448156625032425, -0.1747400462627411, -1.663293719291687, -0.042792994529008865, 0.9456743597984314, -0.5638426542282104, -0.8623204231262207, -0.5851919054985046, -0.829062819480896, -0.5767837166786194, 0.48175203800201416, 0.7933630347251892, 1.4007059335708618, 0.6041545867919922, -0.19055506587028503, 0.8371819257736206, 1.251096248626709, 0.24264827370643616, 0.6573781371116638, -2.0956673622131348, -0.10822713375091553, -0.6414809226989746, 0.5977939367294312, -0.5000418424606323, 0.45504406094551086, -0.20641635358333588, -0.3778601884841919, -0.3129327893257141, -0.24237003922462463, -0.7330251336097717, -0.9843015670776367, -0.026386093348264694, 0.23774085938930511, 0.6784436702728271, -0.4793015122413635, 1.570681095123291, 0.06497981399297714, -0.37245336174964905, -0.31350046396255493, -0.37706637382507324, 1.533678650856018, -0.13555513322353363, -1.6873832941055298, 0.7599238157272339, 1.1774718761444092, -0.595248281955719, 0.12502625584602356, 0.8578765392303467, -0.31731241941452026, -0.7894783020019531, -0.6100114583969116, 2.107875108718872, -0.19303058087825775, 0.4187330901622772, -0.7404711246490479, 1.4753023386001587, -1.4296917915344238, -1.2618013620376587, -1.618111252784729, -0.46408337354660034, -0.4912499189376831, -1.0452566146850586, -0.22015714645385742, 0.6871035695075989, 0.9514214396476746, 0.9149875044822693, 0.08315575867891312, 1.0965501070022583, -0.2743087708950043, 1.1639938354492188, -2.769876003265381, 0.5625745058059692, -0.16317886114120483, -1.056869387626648, -1.2913058996200562, 0.7502613067626953, 0.3055630028247833, -1.7620376348495483, 0.8168143630027771, 0.7723711729049683, -1.4717525243759155, -0.746867299079895, -0.8711653351783752, 1.1101149320602417, 0.7828549742698669, 0.9714459180831909, -0.8983019590377808, 1.0161678791046143, -0.5573601722717285, 0.382241427898407, -0.8280491828918457, 0.0024933938402682543, -0.6914898753166199, 1.096976637840271, 1.901038646697998, -0.30482369661331177, -0.4181496500968933, -0.3108041286468506, -0.48648199439048767, -1.060141682624817, 1.7484230995178223, 0.759529173374176, 1.889359474182129, 0.07128193229436874, 1.1860926151275635, 0.713435173034668, 0.15999749302864075, -0.8494007587432861, -0.4944804906845093, -0.9338062405586243, 0.06855005025863647, 1.3365627527236938, -0.08371905237436295, -1.0037505626678467, 1.4103268384933472, 0.04293721541762352, -0.9317368865013123, 1.984667420387268, 0.006022780202329159, -0.8240267038345337, -0.18770824372768402, -0.7871174216270447, 0.6320507526397705, -0.7915630340576172, -0.9615688323974609, 0.8913049697875977, 0.5583609342575073, 1.1255544424057007, 1.7701977491378784, 0.42644423246383667, 1.023686408996582, 1.255289912223816, 0.11709656566381454, 0.8365507125854492, 0.6054449677467346, -0.9390354156494141, 0.31269732117652893, 0.5342097282409668, -0.5443100333213806, 0.7418738603591919, 0.08316964656114578, -0.02209213748574257, 0.7111260294914246, -0.0952715203166008, 2.016998291015625, -1.2549580335617065, 0.4917491674423218, -0.9876230359077454, -1.502798318862915, -0.20094184577465057, 0.5836751461029053, -0.9646257162094116, -1.6841163635253906, 0.66005939245224, 2.10783314704895, 0.8346686363220215, -0.4440714418888092, -0.31334543228149414, -0.6339675188064575, 0.7838153839111328, -1.1942344903945923, 0.3822808265686035, 0.15193423628807068, -0.5012284517288208, 1.5148106813430786, 0.40522128343582153, 0.20867976546287537, 1.115073800086975, -1.2724623680114746, 0.4132859706878662, -1.655066728591919, -1.89699125289917, -1.172831654548645, 1.0787622928619385, -0.5330622792243958, 1.046690583229065, -0.19249063730239868, 2.071039915084839, -0.7751089334487915, -0.5337023138999939, -0.3675820529460907, -0.7466557621955872, -0.10137190669775009, 1.8736714124679565, -1.4864782094955444, -0.04053080826997757, -0.07245989143848419, 0.945717990398407, -0.965747594833374, 0.928187906742096, -0.6743333339691162, -0.23624837398529053, -0.16012713313102722, -1.0104893445968628, -0.23408079147338867, 0.6446107029914856, 0.8588594794273376, -1.0569233894348145, 0.42638421058654785, -0.05507994443178177, 0.904840886592865, 1.8576030731201172, 0.2823214828968048, -0.4661914110183716, 0.40723854303359985, 0.40312835574150085, 0.524960994720459, 0.5749394297599792, 0.06391917914152145, -0.30497029423713684, 0.4742957651615143, -0.13350529968738556, 0.1713293343782425, 2.1129636764526367, 1.1280525922775269, -0.2705356478691101, 0.43243861198425293, -0.1960608810186386, -0.5624713897705078, -1.0770760774612427, 0.4526909589767456, -0.27053019404411316, 0.4830087721347809, -1.0108155012130737, 1.3954182863235474, -0.42499321699142456, 0.28493034839630127, -0.00975255761295557, -3.1386258602142334, 0.7100043296813965, -0.4366585910320282, -0.8916860222816467, 0.47105276584625244, -1.1563912630081177, 0.17590945959091187, 0.5567935109138489, 0.9461489319801331, 1.1150518655776978, -0.36686110496520996, -0.4793398082256317, 0.8078529238700867, -1.1731361150741577, 0.49700427055358887, 0.5291484594345093, -1.7023546695709229, 0.18876294791698456, 1.1683732271194458, 0.03527018427848816, 0.483220636844635, -0.19168895483016968, 1.0852686166763306, -1.3161894083023071, -2.6186442375183105, -0.7064722776412964, -0.003924536053091288, 0.4723483622074127, -2.2046573162078857, -0.6497318148612976, -0.5235285758972168, 0.9066085815429688, 0.595594584941864, 0.49905455112457275, 2.421298027038574, 2.2365036010742188, 1.1857613325119019, -2.1883888244628906, -0.5784037113189697, -2.1066951751708984, -0.5456036329269409, 0.332985520362854, -1.063854455947876, 0.9878848195075989, -0.19354620575904846, 1.0338363647460938, 0.14083389937877655, -1.084449291229248, 1.3089478015899658, -1.4857406616210938, -1.1338002681732178, 1.0683671236038208, -0.15525370836257935, 0.30824923515319824, -0.3030955493450165, -0.6937339305877686, -1.5314764976501465, 0.7356353998184204, 1.017983078956604, -0.16437898576259613, -0.2947652339935303, 0.8849761486053467, -1.4962836503982544, -0.34484025835990906, 1.5473524332046509, 0.22755762934684753, 1.2658377885818481, -0.20060406625270844, -0.4883538484573364, 0.08468108624219894, -0.061078671365976334, -0.03952151536941528, 0.5865839719772339, 1.3096407651901245, -1.2593461275100708, -0.38947781920433044, -0.2662402093410492, -0.29915937781333923, 0.9157175421714783, -0.23458197712898254, -1.1578396558761597, -0.8514638543128967, -0.7381569743156433, -0.8954095840454102, -0.3946029841899872, -3.1403026580810547, 0.2583799958229065, 0.7562066912651062, 0.16438913345336914, 0.2981015145778656, -1.6673952341079712, 0.18542547523975372, -0.048007782548666, -0.10243069380521774, -1.919180989265442, -0.720258891582489, 0.307950884103775, 1.927223563194275, -0.010534228757023811, 1.0443642139434814, -1.7846087217330933, 0.2498432695865631, 0.9614304900169373, 1.3232532739639282, 1.3720897436141968, -0.6112096905708313, 0.3756261467933655, -0.11222165077924728, 0.14613328874111176, 0.6103403568267822, 0.4941205680370331, -0.17085886001586914, -0.8152959942817688, -1.5080974102020264, -1.5356944799423218, 1.305529236793518, 0.20977438986301422, 0.5075126886367798, 1.7787772417068481, 0.6363085508346558, -0.7889668345451355, -0.8526017069816589, 0.6059049367904663, 1.0575505495071411, -0.31012803316116333, -0.11205029487609863, 0.8078479766845703, 1.3068945407867432, 0.9218181371688843, -0.6725379824638367, 0.09025917202234268, 0.3132092356681824, 0.7385586500167847, -1.1765968799591064, -0.3101919889450073, 0.14980728924274445, -1.6521658897399902, -2.1475977897644043, -0.18916040658950806, -0.16630378365516663, 0.956203281879425, -0.1296168714761734, 0.20321644842624664, 0.37574872374534607, -0.23841769993305206, -0.1572147160768509, 0.1960703730583191, -1.9776499271392822, 1.438205361366272, 0.0012074782280251384, 0.09836617112159729, -0.595453679561615, -0.9791212677955627, -0.8589543700218201, 0.19791406393051147, -0.06761375069618225, -0.8725399971008301, 0.7402382493019104, -1.5374723672866821, 0.3732011616230011, -0.3228554129600525, 0.8737740516662598, -1.5618455410003662, -0.6848052740097046, -0.675510048866272, -1.636957049369812, -1.4422551393508911, -0.0961538627743721, -0.41095295548439026, -2.212986469268799, -0.312085896730423, 2.272712469100952, 0.0756411999464035, 1.0460827350616455, -1.665600299835205, 0.5721181631088257, 0.28655800223350525, 0.5814282298088074, -0.8719560503959656, -0.7616409063339233, -0.5560441613197327, 0.7666398882865906, -0.34932106733322144, 0.8649181127548218, -0.29744285345077515, -0.7349597811698914, 0.9591972827911377, -0.3325442969799042, 1.2363667488098145, 0.33000820875167847, 0.45697152614593506, -0.5952625870704651, -0.2619260549545288, -1.752129316329956, -0.1160571277141571, 0.2909087538719177, 0.2827187776565552, 0.8610588908195496, -0.3827415406703949, -1.4669767618179321, 0.1021459624171257, -0.5438435673713684, -1.2093448638916016, -0.01586362160742283, -0.24499933421611786, 0.3126993179321289, 0.8908519148826599, 1.5670970678329468, 0.5488126277923584, -0.28228959441185, 0.8641465902328491, -1.6279151439666748, 0.7342387437820435, -1.528693437576294, -1.0548125505447388, -0.021733732894062996, 0.1566700041294098, -1.1017314195632935, 1.2944247722625732, -0.3118622899055481, 0.19403432309627533, 0.9408056735992432, 0.8073091506958008, 1.1432886123657227, 1.1631286144256592, -0.9111976027488708, 1.146109700202942, 0.1632513850927353, -0.6055102348327637, -1.8311458826065063, -0.49106937646865845, -0.03720353543758392, 0.5466578006744385, 0.5858879685401917, 0.6363037824630737, 0.2453085035085678, -0.7411762475967407, -1.147349238395691, -0.6328967809677124, -0.5411592721939087, -0.11442532390356064, -0.00821747351437807, -1.9080482721328735, -0.26200011372566223, 0.41197794675827026, 0.4470808506011963, 0.1720125377178192, -0.6952779293060303, -0.6881990432739258, 0.3364565968513489, 1.3107465505599976, 0.15772579610347748, 0.12965957820415497, -1.1763052940368652, 0.41218432784080505, 1.8174161911010742, 1.398464560508728, 1.3172141313552856, -1.5016446113586426, -0.26599958539009094, 0.09791883081197739, -0.6256298422813416, -0.11431178450584412, -1.8438090085983276, 1.92145836353302, 0.26069849729537964, -1.338067889213562, 0.2393634170293808, -0.21449661254882812, -1.730669617652893, -0.3808411657810211, -0.1919376254081726, 0.8112621903419495, 0.5868936777114868, 0.6727450489997864, -0.4863080680370331, 1.8079684972763062, 0.8188982605934143, 0.09947226196527481, -1.1842656135559082, -0.64417964220047, 0.4756954610347748, 1.0714839696884155, -0.3440982699394226, -1.805983543395996, -1.043518304824829, -0.37432804703712463, 0.991097629070282, -0.9247588515281677, 2.3390989303588867, 0.17707884311676025, -0.5766146183013916, -0.5861037969589233, -0.5769453048706055, -0.25785139203071594, 0.28945839405059814, 0.4682147204875946, -1.1631823778152466, -0.5358993411064148, 2.065391778945923, -0.6894469857215881, 0.044631801545619965, 0.11398189514875412, -0.59053635597229, 0.9177821278572083, -0.3475203812122345, -0.696180522441864, -1.2717012166976929, 0.5120403170585632, 0.7425954341888428, 1.0720875263214111, -0.022792581468820572, -1.2609597444534302, -0.7251799702644348, -1.2118005752563477, 0.6173693537712097, -1.2884844541549683, -1.6042144298553467, -1.1596168279647827, -0.32216617465019226, -0.8090204000473022, -0.11138202250003815, 1.0557665824890137, -0.1636064201593399, -0.9659748077392578, 0.7130533456802368, -0.8366473913192749, 0.5589631795883179, 0.34508833289146423, 0.6614177823066711, -0.23407700657844543, -0.1479426920413971, 0.7339503765106201, -0.48446738719940186, 1.1932154893875122, -0.007550982292741537, -0.9822742938995361, -0.7258617877960205, -0.18919925391674042, 0.15271686017513275, -0.7164987325668335, -0.3358219861984253, -0.5581791400909424, -0.9754022359848022, 1.4233081340789795, 0.11537430435419083, -0.7277413606643677, 0.2075759470462799, 0.7494686245918274, -1.5661972761154175, 0.20261242985725403, 1.0224494934082031, 0.7260743379592896, 1.2075510025024414, -0.17405912280082703, 1.0286953449249268, 1.4560712575912476, -0.32288575172424316, 0.3555263578891754, 0.504655122756958, 0.08284103125333786, 0.04300881549715996, -0.023054860532283783, 0.18634390830993652, -1.3143842220306396, 1.517643928527832, -1.3549704551696777, -0.9992358684539795, 0.1637558937072754, -1.9660969972610474, 0.3385125696659088, 1.666174054145813, -0.7404653429985046, 0.7269906401634216, -2.006492853164673, 1.9879158735275269, 0.34713229537010193, -2.0641865730285645, 1.651599645614624, -1.2340670824050903, 0.4110057055950165, -1.5639842748641968, -0.32133349776268005, 0.7741835117340088, -0.7784367799758911, -1.2565194368362427, 2.4436802864074707, 2.8115475177764893, -1.636869192123413, -1.1004408597946167, -0.8306759595870972, -0.6296494007110596, -1.7297358512878418, 0.8821850419044495, 0.6135115027427673, -1.0671215057373047, -0.12899456918239594, -0.8345120549201965, 1.138739824295044, -0.8616366982460022, -1.183185338973999, 0.4657377302646637, -2.0230157375335693, -0.22653627395629883, -0.8043884038925171, 0.342245876789093, -0.7356572151184082, -1.1566286087036133, -0.4630025625228882, -1.6423966884613037, 0.916099488735199, -1.3085248470306396, 0.6065971851348877, 1.26985764503479, -0.13340969383716583, -0.4670799970626831, 0.03244781866669655, -1.312052845954895, 1.0431381464004517, -1.2044036388397217, -1.6600815057754517, -0.5698022246360779, -0.306744784116745, -0.6063827872276306, -0.34577512741088867, -1.5941014289855957, 1.4315528869628906, -1.0785547494888306, -0.43934348225593567, -0.2862241864204407, -0.33192265033721924, 0.2753937542438507, 1.4851411581039429, -1.8122739791870117, -0.7096453905105591, -0.81524258852005, 0.3542404770851135, 0.8802616596221924, -0.4560236930847168, -0.15480774641036987, 1.438897728919983, 0.9247260689735413, 0.7801960706710815, -1.1688355207443237, 0.6362119317054749, 0.20769861340522766, 0.6318837404251099, -0.4854954481124878, -0.20626141130924225, -1.3270342350006104, -2.1100287437438965, -0.17826266586780548, 0.4096870422363281, -0.22571049630641937, -1.4461325407028198, 1.7094272375106812, 0.7868101596832275, 2.113626718521118, -0.23297622799873352, 0.8402455449104309, 0.22518810629844666, -1.5335890054702759, -0.8230178356170654, 0.5464675426483154, -0.9222293496131897, 1.9192668199539185, -1.2965914011001587, -0.39363953471183777, -1.2039422988891602, -1.0545777082443237, 1.292222261428833, 1.4084550142288208, 1.6378834247589111, 0.4890323281288147, -0.3176746368408203, 0.16463249921798706, -1.2718448638916016, -0.3159181773662567, 0.2554721236228943, -0.40582185983657837, 2.0454225540161133, -0.06490227580070496, 0.16257208585739136, 0.5406899452209473, 1.4378762245178223, 0.08287665247917175, 0.5458349585533142, -0.9316844344139099, -1.129281997680664, -0.4525051712989807, 0.6924667358398438, -1.1627390384674072, -1.6302058696746826, 1.5794451236724854, 0.8321049809455872, 1.331910490989685, -1.2196204662322998, -1.1902351379394531, -0.2638758420944214, 1.0998026132583618, -2.2447075843811035, 1.9849376678466797, -0.25602638721466064, 1.4106998443603516, 1.7273688316345215, 0.5756895542144775, 1.0531244277954102, -0.9648179411888123, -0.6339163780212402, -0.7942713499069214, -0.4585694372653961, 1.2606570720672607, 0.11341787874698639, 1.2972806692123413, 0.09587237983942032, -0.9954245686531067, -0.8359984755516052, 2.7181310653686523, 0.691288948059082, 0.14850273728370667, -0.6003252267837524, -0.2099764198064804, 0.0870722308754921, 0.6105910539627075, 0.04639236629009247, 1.5970405340194702, -1.1416120529174805, -0.027103746309876442, 2.0577569007873535, -0.6085034608840942, 0.39313462376594543, 0.9600736498832703, -0.08530674129724503, -0.7470259070396423, -0.8237984776496887, -0.4119020700454712, 1.7185089588165283, 0.05002404376864433, 0.7194754481315613, 0.897956132888794, -0.09780064225196838, -0.26074469089508057, 0.5424817800521851, 0.9853000044822693, -1.0299394130706787, -0.5707763433456421, -0.652305006980896, -1.7753665447235107, -0.7948888540267944, 0.009770860895514488, -1.6139036417007446, -0.5349546074867249, 0.9748477935791016, 0.006227550096809864, -0.6395313739776611, 0.9203718900680542, 0.06935491412878036, 0.30810800194740295, -0.1277734488248825, 0.6199594140052795, 0.40001699328422546, 0.012211873196065426, 0.3576899766921997, 0.09472494572401047, -0.6875860691070557, -0.14035020768642426, 0.4901241064071655, -0.8873517513275146, 1.041859745979309, -0.11208728700876236, 0.12932635843753815, 0.992122232913971, -0.44640615582466125, 1.2369827032089233, 0.49255073070526123, -0.08863583207130432, -1.1171491146087646, 0.7433570623397827, -0.25553447008132935, 0.7011019587516785, 1.0461422204971313, -0.08598306030035019, 0.7029294967651367, 0.40068960189819336, 0.6719003915786743, -3.0847535133361816, 0.7234988808631897, 0.44738468527793884, -1.1847280263900757, 1.1820690631866455, -1.8060169219970703, -0.2658676505088806, -1.2712492942810059, 0.43944600224494934, -0.2915353775024414, -0.26028183102607727, -0.04186311736702919, -0.07956022024154663, 0.29772692918777466, 0.7749704122543335, -1.773538589477539, 0.029972679913043976, 0.22304393351078033, 1.2940244674682617, 2.0680601596832275, 1.664398193359375, -0.7255547046661377, -0.312203586101532, -0.45233815908432007, 2.8254199028015137, -0.7646694779396057, -0.05940936133265495, 1.0825786590576172, 0.15378710627555847, 1.2855356931686401, 0.2730717360973358, -1.002369999885559, -1.16218900680542, -0.13216465711593628, -0.2427709400653839, -0.2812114953994751, 0.8992035984992981, 0.545623242855072, 0.3510449528694153, -2.32690167427063, -0.47931182384490967, -0.8492889404296875, 0.309826523065567, -0.29613110423088074, 0.41832107305526733, 1.460415244102478, -1.7333780527114868, 0.3601468801498413, -1.3294578790664673, -0.16180916130542755, 0.34157341718673706, -0.6821791529655457, -0.4798237085342407, 0.12818631529808044, -0.17849141359329224, -2.1152637004852295, 0.818642258644104, -1.4338419437408447, -0.9073707461357117, -0.9147263169288635, -0.9373217225074768, 0.6136392951011658, -0.41137823462486267, -0.6372761130332947, 0.7796814441680908, 1.2052313089370728, -0.3415462076663971, 1.440528392791748, 0.5357574224472046, -0.07444748282432556, 0.39773139357566833, 0.11996039003133774, 0.4123609662055969, -0.06611400097608566, -0.6730955839157104, -0.42704781889915466, 0.07951797544956207, 1.4063438177108765, 0.601302981376648, -1.8309686183929443, -0.4139494299888611, 0.5519109964370728, -0.7986673712730408, 0.10241927951574326, -0.3900836110115051, 0.15340641140937805, -0.15181700885295868, -0.4463135600090027, 0.339009553194046, -0.7732347249984741, 0.4677644670009613, -0.33371689915657043, -0.3094026744365692, -0.26222461462020874, -2.308138847351074, 0.8387052416801453, -0.04314319044351578, 1.2718433141708374, 0.19245053827762604, -0.22683362662792206, 0.06207113340497017, 1.1841304302215576, 0.8202620148658752, 0.4822984039783478, 1.0980147123336792, 0.7001399993896484, 1.3301438093185425, -0.34202468395233154, -0.9383918046951294, -0.0405607670545578, 1.8503565788269043, -1.287029504776001, -0.9048593044281006, 1.6380469799041748, 0.7171326279640198, 0.3786587119102478, -1.439864993095398, 1.0245513916015625, 2.3187663555145264, -0.4344286620616913, -0.9654132723808289, -0.19128882884979248, 0.9560630917549133, 1.437997579574585, -0.20077605545520782, 2.0635805130004883, 0.3419052064418793, -0.37274476885795593, -0.751790463924408, 0.14811348915100098, 0.006337046157568693, -1.5845974683761597, 0.8236908912658691, -0.8004439473152161, -0.7885300517082214, 0.298335462808609, 0.6912591457366943, -0.5763977766036987, 0.7194003462791443, -0.9900454878807068, 0.6349846124649048, -0.7698335647583008, 1.0707581043243408, -1.038602352142334, 2.225475788116455, -0.06511478871107101, -1.075219750404358, -1.28374445438385, -0.047585006803274155, -0.4988502562046051, -0.08133772015571594, 3.1569838523864746, -0.1489943265914917, 0.8297939300537109, 0.23233187198638916, 0.3685198128223419, 0.41160884499549866, -0.03106139414012432, 1.2893998622894287, -0.08112619072198868, -0.16458405554294586, 1.5147039890289307, -0.6077685356140137, 1.506649136543274, -0.6488917469978333, -1.1402369737625122, 1.500435471534729, 0.3482094407081604, 0.459427148103714, 0.5385192036628723, 0.3974156975746155, 0.3954452872276306, -1.201621413230896, 1.097436785697937, -0.7101357579231262, -0.1713387370109558, -0.400438517332077, 0.7145797610282898, -0.49865809082984924, 1.180579423904419, -0.6277440190315247, -0.03558177500963211, 0.09558682143688202, 1.218359112739563, -0.5402832627296448, 0.867805004119873, -1.6041642427444458, -1.5336867570877075, 0.7902398705482483, -0.19477519392967224, 2.1980812549591064, -0.11305247992277145, -0.8696735501289368, 1.1953375339508057, 0.007401651237159967, 1.724730372428894, -0.7447795271873474, 2.699063539505005, -0.17015904188156128, 1.5568945407867432, 0.6746405959129333, 0.3964560329914093, 0.10002709180116653, -0.3665274679660797, -0.4682430326938629, 0.32386675477027893, 1.0822701454162598, 0.2596310079097748, 0.40764033794403076, -0.049676794558763504, -1.1375236511230469, -0.1491595059633255, 0.09916527569293976, 1.0094836950302124, -1.914393663406372, -0.37003231048583984, 1.5404210090637207, 0.046688444912433624, 1.3313415050506592, 0.43030139803886414, -0.2678976058959961, 0.4387443959712982, -0.14265750348567963, -1.6243712902069092, -1.3168296813964844, -0.011328499764204025, 0.4998035728931427, 1.0649584531784058, -0.184316948056221, 0.8046609163284302, 0.4289616048336029, -0.16021375358104706, -0.053583405911922455, -0.7782765030860901, -0.025512009859085083, -0.24628984928131104, -0.06292062252759933, -0.2870056927204132, -0.7960376143455505, 0.3720491826534271, -0.7202090620994568, 2.367309331893921, 0.5466642379760742, -0.8931624889373779, -0.5960583090782166, -2.175184726715088, 1.2549619674682617, 0.39269745349884033, -2.0801467895507812, -0.626944899559021, -1.2615435123443604, -1.398447871208191, -0.11697268486022949, 0.5207398533821106, -2.0185022354125977, -0.5800189971923828, -0.1009158045053482, -0.019910991191864014, -0.4445103704929352, -1.2674356698989868, 1.0075092315673828, -2.299804210662842, 0.6789677143096924, -0.8047887682914734, -0.670913815498352, 0.4072611927986145, 0.33032625913619995, 0.7780881524085999, 0.9201725125312805, 2.0978705883026123, 1.3022514581680298, -0.020197639241814613, -0.6561828255653381, 0.40028518438339233, -0.5978018045425415, -0.46921882033348083, -0.4060700535774231, 0.048483435064554214, -1.0656934976577759, -0.10825926810503006, -0.1911032646894455, -0.7458849549293518, 0.5111226439476013, 0.09035731106996536, -0.23001334071159363, -1.440220594406128, -0.47412070631980896, -0.6785915493965149, -0.784456729888916, 0.8304593563079834, 0.06885755062103271, 0.17731262743473053, -1.050285816192627, 0.2096930295228958, -1.5862462520599365, 0.39199960231781006, -0.5175211429595947, 1.6694940328598022, -0.8843429088592529, 0.0408150814473629, -1.0883771181106567, -1.5298616886138916, 0.948392927646637, -1.128027319908142, 1.6171596050262451, -1.820120930671692, 0.6821619272232056, 1.300767183303833, -0.09241154789924622, 1.042873501777649, -0.30909091234207153, -1.3913406133651733, 0.4262188673019409, -1.0688450336456299, 0.15618683397769928, -0.34916651248931885, -0.4445316791534424, 0.9588321447372437, -0.40689244866371155, -0.6145384907722473, -0.7570763826370239, 1.9983566999435425, 1.7482744455337524, -1.0888276100158691, 0.9386390447616577, 0.5600239634513855, -0.4248059391975403, -0.9710782766342163, 0.03851454332470894, -0.43756726384162903, -0.7086721658706665, -1.132055640220642, 0.4101698100566864, 0.7476674318313599, 0.7017131447792053, 0.11643202602863312, 0.976026713848114, -0.5307645797729492, 0.3516165316104889, -0.3505072593688965, -0.9083892107009888, -1.6068971157073975, -0.3321676552295685, -1.1598200798034668, -0.040590520948171616, -0.6355006694793701, -0.5888372659683228, 0.2010616958141327, 0.39730989933013916, -1.046297550201416, -0.08901197463274002, -0.20442454516887665, -1.0668880939483643, -0.3928599953651428, -0.10488800704479218, -0.13934718072414398, -0.1555621176958084, -1.833417534828186, 0.1791795790195465, -1.7292853593826294, 1.8616660833358765, -0.06897921115159988, -1.3632779121398926, 0.03658642619848251, -0.8087212443351746, -3.147473096847534, 1.0772147178649902, -0.43840575218200684, 0.5888518691062927, -0.2917172610759735, -0.7117149829864502, 0.6272274255752563, 0.7626801133155823, 0.9638285636901855, -1.2211843729019165, 1.2422714233398438, -0.4070957601070404, -0.6317825317382812, 0.7025715112686157, -0.25903141498565674, 0.058794599026441574, -0.9192467927932739, 0.14277416467666626, 0.764618992805481, 0.9494898319244385, 0.12380225956439972, -0.2832741439342499, -0.39166292548179626, -0.3742332458496094, -1.0225539207458496, 0.14751067757606506, 1.2801917791366577, 0.44795364141464233, 0.2671652138233185, -0.1620539426803589, 0.606022298336029, -0.9674599170684814, -1.521241307258606, 1.8602192401885986, -0.45074358582496643, 1.3440680503845215, 0.11032271385192871, -0.8142326474189758, -0.27784794569015503, -0.09855920821428299, 0.188307523727417, 1.6068476438522339, 1.6354295015335083, 0.5391663908958435, 1.2162495851516724, -0.8385022878646851, -1.0676915645599365, 0.8959577083587646, -0.6770842671394348, -0.4971733093261719, -0.09502044320106506, 0.07002046704292297, 0.9535443782806396, 0.7867904305458069, -0.8824975490570068, 0.13559556007385254, 0.4736826419830322, 0.07415857166051865, -0.7466434836387634, -0.22266404330730438, -2.6330406665802, 0.08702229708433151, -0.14735610783100128, 0.5207609534263611, 1.5209598541259766, 0.5504130721092224, -2.073604106903076, 0.8952171206474304, 0.7223214507102966, -1.2227481603622437, -0.6167500019073486, 0.9796234369277954, -0.8249156475067139, -1.0443059206008911, 2.4420626163482666, 1.6081640720367432, 0.7619712948799133, 0.36193111538887024, 1.0599490404129028, 0.221269428730011, -0.20826290547847748, -0.3698378801345825, -0.4718593955039978, -0.3481191098690033, -0.051222432404756546, 0.14198991656303406, -1.100128173828125, 2.4171950817108154, 1.144474983215332, -0.20311623811721802, -2.439908981323242, -1.7781966924667358, 0.42296329140663147, -0.7681930065155029, -0.05638473480939865, 0.5503803491592407, 1.7619379758834839, 0.36233028769493103, 0.10195425152778625, -0.738109290599823, -0.6942726373672485, -0.44500136375427246, 0.31261977553367615, -0.3577215075492859], [-1.0015850067138672, 1.6862598657608032, 1.8591248989105225, 1.3350352048873901, -0.18784581124782562, 0.8471863865852356, -2.1206891536712646, 0.10183962434530258, -0.7010325193405151, 0.953291118144989, 0.2482256144285202, 0.30492228269577026, 1.0381090641021729, -0.12684524059295654, -0.10067547857761383, 0.6399572491645813, 0.005969315301626921, -0.419735312461853, 0.2333606332540512, -0.9167275428771973, -1.0645521879196167, -0.10793567448854446, -1.1267499923706055, -0.8872592449188232, 0.4263460338115692, 0.9539874792098999, -0.6523800492286682, 1.6024527549743652, -1.7740302085876465, -1.2912110090255737, 0.5580291748046875, 1.6379613876342773, -0.4208858907222748, -1.5337343215942383, -0.715497612953186, -0.36367884278297424, -1.122850775718689, -0.6626002788543701, 1.109212875366211, 1.4148354530334473, -0.7444428205490112, 0.7191582322120667, 1.5782248973846436, 0.507032573223114, -0.5322193503379822, 1.5384306907653809, 1.1572169065475464, -0.6103771924972534, -0.30545467138290405, 0.39818572998046875, 1.458125352859497, 0.9944756031036377, 1.3146635293960571, 0.7684144973754883, -1.2326226234436035, -2.242600440979004, -0.7823355793952942, 0.48970383405685425, -0.33910492062568665, 0.4637788236141205, 0.12185171991586685, 0.21178190410137177, 0.08561569452285767, -0.8794935345649719, 1.4592136144638062, 0.8085546493530273, -0.1946694254875183, 1.262037754058838, -0.32587942481040955, -0.3426429033279419, 1.9310818910598755, -1.0868098735809326, -0.48498955368995667, 1.2465062141418457, 0.08513465523719788, -1.5245416164398193, 0.343917578458786, -1.5686813592910767, 1.282713532447815, 1.179840087890625, 0.5937638878822327, 0.04065486043691635, -1.098413109779358, -0.6480870842933655, -0.20301827788352966, 0.38995614647865295, -0.04394896328449249, 0.6575700044631958, 0.33902814984321594, -0.7773433923721313, -1.0789343118667603, -0.4193056523799896, 1.9953303337097168, -0.764518141746521, -0.2071353942155838, -0.6894745826721191, -0.05315147712826729, -0.6556375622749329, -0.46262606978416443, 1.120219111442566, 0.48484107851982117, -1.6815625429153442, -0.5141850113868713, 1.3962141275405884, -1.113165020942688, -0.5859361290931702, -0.09717836230993271, -0.9411860704421997, -0.19504044950008392, -1.0609723329544067, -0.1609726995229721, 0.8815608620643616, 2.2742655277252197, 1.6395466327667236, -0.23139332234859467, 0.6038656830787659, -2.1800572872161865, -0.7845020890235901, -0.45836877822875977, -0.6534435749053955, -0.2660176753997803, 1.6036735773086548, -0.03748257830739021, -0.2723778188228607, -1.95708167552948, -0.3593193590641022, -0.01029886119067669, 0.14252279698848724, 0.6344817280769348, 0.785182774066925, 0.5335347652435303, -0.40695616602897644, 1.60702645778656, -2.167888641357422, 0.10818994790315628, 1.8291057348251343, -1.0905112028121948, 0.27516499161720276, -0.04616527631878853, 0.44093453884124756, 0.9744990468025208, -0.2968660891056061, 0.43588972091674805, -0.4073267877101898, -1.3889025449752808, 0.03447149321436882, 0.9223219752311707, 0.21457412838935852, 0.24334768950939178, 1.6621217727661133, 0.352069616317749, -0.04781041666865349, -0.2135835736989975, -0.4562373757362366, 1.5672028064727783, 0.6383653283119202, 0.8258607983589172, -0.7765404582023621, 0.7662233710289001, -0.3209962546825409, -0.1939418762922287, 0.12914705276489258, 0.055396243929862976, -0.12489910423755646, -2.101576089859009, -0.3931981027126312, -0.9478120803833008, -1.1541870832443237, 0.9719904661178589, 1.3782916069030762, -0.3752460479736328, 0.8696995377540588, -0.33230510354042053, 1.6950852870941162, -0.5516993403434753, -1.2872190475463867, -0.007072944659739733, -0.046438299119472504, -0.5706024765968323, -0.2257002294063568, -0.08336478471755981, 1.4058815240859985, -0.0027773831970989704, -0.7086669206619263, 0.268353670835495, -0.1934828907251358, -1.971518635749817, -0.13579800724983215, 0.2302086502313614, 0.16774916648864746, 0.4448641240596771, -1.2576202154159546, -0.022902218624949455, 0.07293139398097992, -0.08928846567869186, 0.826302707195282, -0.2871442437171936, 0.0673593208193779, 0.2010609656572342, 2.0524682998657227, -0.30550533533096313, -0.36721792817115784, -0.6029022336006165, -0.5823249816894531, 0.8607767820358276, -0.36718979477882385, 1.0404856204986572, -0.3421981632709503, -1.402847170829773, -0.15442074835300446, -2.8386070728302, -0.11655072122812271, -0.30773812532424927, 1.1338768005371094, 0.15620584785938263, -1.4261194467544556, 0.3795566260814667, -1.2530378103256226, -0.07912182807922363, 0.37986770272254944, -0.44216907024383545, 0.2652237117290497, 0.1145089790225029, 0.08570022881031036, 1.1945257186889648, 1.245755910873413, -0.34970623254776, 0.7572149634361267, -1.9830440282821655, 0.2672467827796936, 0.5748947262763977, -0.5832275748252869, 0.10285864025354385, -0.1774589866399765, -0.6734539270401001, 1.9080169200897217, 1.9953672885894775, 1.1572223901748657, 1.354402780532837, -1.163049578666687, -1.0736243724822998, 0.6160737872123718, -1.2752571105957031, -0.9423052072525024, -0.5568415522575378, -0.5312264561653137, -0.7399535179138184, 1.4473624229431152, 0.3371466100215912, -0.10374061018228531, 1.0857529640197754, -0.304988294839859, -1.087889552116394, 1.3855617046356201, 0.10639443248510361, -0.9030419588088989, 0.6534319519996643, 0.2640654444694519, -1.0017846822738647, 0.30412915349006653, -0.6010235548019409, -1.622377634048462, -0.22568224370479584, -0.4492805600166321, -0.4948498010635376, -0.6191124320030212, -0.9719083905220032, 0.08846800774335861, 0.91487717628479, -0.9103933572769165, 0.03377305343747139, -1.0315085649490356, -0.4207114279270172, 0.5132988691329956, 0.14282746613025665, 0.7704818844795227, 0.47054189443588257, -2.452376365661621, -1.6385899782180786, 0.16729256510734558, -2.1007251739501953, 1.1699808835983276, 0.7836326956748962, 0.42913487553596497, 0.3928427994251251, -1.991791009902954, 0.28556016087532043, -0.9077772498130798, -0.5264652371406555, 0.774246871471405, -0.14045080542564392, 0.7879547476768494, 0.1909785121679306, -0.09630318731069565, 0.7677037119865417, 1.053238034248352, -0.9944824576377869, -1.3179682493209839, -0.013619625940918922, 0.07258100062608719, 0.5394963622093201, 0.44540008902549744, 0.9037668704986572, -1.612251877784729, 0.4720054566860199, -0.745570957660675, 0.704544186592102, -0.44373977184295654, 0.37248119711875916, 0.3329927623271942, 0.4953997731208801, -0.3234691023826599, -1.364949107170105, 2.131629228591919, 0.5842700600624084, -1.0604480504989624, -0.6845114231109619, 1.2039315700531006, -0.005602188408374786, -0.2855920195579529, -0.621773898601532, -0.40131157636642456, -0.33125147223472595, 1.3374743461608887, 0.5741623640060425, -0.3799022138118744, -1.1834698915481567, -0.5225549340248108, 0.5327121615409851, -0.8798890113830566, -0.015487141907215118, -1.1646956205368042, -1.2590488195419312, 0.6767770051956177, 2.0943658351898193, -0.4020889103412628, -0.21593202650547028, -0.15620838105678558, -0.7337930202484131, 0.13911771774291992, -0.34449303150177, 0.892150342464447, 2.270145893096924, -1.0047818422317505, 0.3108249008655548, -0.20116448402404785, 0.29697248339653015, 0.8072236776351929, -0.27434980869293213, -0.4969254434108734, 0.053735241293907166, 0.21130135655403137, 1.0514981746673584, 0.2635907232761383, -0.07803145796060562, -0.8743468523025513, 0.03364339470863342, 0.34137749671936035, 0.8116734623908997, -1.827630877494812, -0.7756872177124023, 0.5496753454208374, -0.9629698395729065, 0.31690514087677, 0.3762161135673523, -0.9062685370445251, 0.3296528160572052, -1.4425772428512573, 0.0195330660790205, 1.1058149337768555, -1.1852402687072754, 1.3793041706085205, -0.16487827897071838, -0.2734980881214142, 0.05446967855095863, 0.7537052035331726, 0.8197842836380005, -0.3933603763580322, -0.4968329071998596, 1.2390559911727905, -0.8615432381629944, -1.0188316106796265, -0.15087169408798218, -1.2106575965881348, -0.6608087420463562, -0.6104697585105896, -1.65215003490448, -1.094225525856018, -0.6468597054481506, 0.6826081871986389, 0.03649575635790825, 1.4235692024230957, -1.540556788444519, -0.9407926797866821, 0.6209815740585327, -0.525564968585968, -0.2206496298313141, 0.7736586928367615, -0.681481122970581, -2.189927816390991, 0.45697417855262756, 2.6414895057678223, 0.11642199754714966, -0.18257103860378265, -0.19742102921009064, -0.4728357791900635, 0.3559921383857727, -0.07406509667634964, -1.1035715341567993, -0.9959363341331482, -1.0542175769805908, 0.27219516038894653, -0.048330482095479965, -1.343727469444275, -0.8510695099830627, 0.5826545357704163, -0.7031744122505188, 0.3833511471748352, -1.5851037502288818, 0.4095149338245392, -0.3617694675922394, 1.14878249168396, -1.3125234842300415, -0.572312593460083, 0.5846042037010193, 0.6690528988838196, 0.39699453115463257, -0.295089453458786, -0.16769787669181824, -0.28230905532836914, 0.6226038932800293, 0.8616473078727722, -1.0692905187606812, 1.0893125534057617, 0.20197808742523193, -0.5713686347007751, -0.7218409776687622, -0.1664370447397232, -0.43744558095932007, -0.20743009448051453, 0.127966046333313, -0.6902344822883606, -0.27221688628196716, 0.9405724406242371, -0.9173130989074707, 0.5391865372657776, -1.247575044631958, -1.077846646308899, -0.005203901790082455, 0.34285247325897217, 1.7434864044189453, 0.5334979295730591, 0.4776477813720703, 1.386149525642395, -0.2674510180950165, 1.147807240486145, -0.5679663419723511, 1.1786848306655884, 1.5419232845306396, -0.18613868951797485, 1.3225743770599365, 1.1066535711288452, -0.9158740043640137, 0.38017967343330383, -0.5104151964187622, 1.4117846488952637, 2.1652588844299316, 0.7508301734924316, -0.8234401941299438, 0.608127772808075, -1.1323096752166748, -0.21140830218791962, 0.5745202302932739, 0.8391517996788025, -0.5330402255058289, -0.8780681490898132, 0.12349684536457062, 0.3203687071800232, -1.1932122707366943, 0.6320854425430298, -0.6290944218635559, -0.33093729615211487, 0.44836869835853577, 1.1893796920776367, -0.6976255774497986, -2.146514415740967, 1.1611886024475098, -2.2768921852111816, 0.9347610473632812, -0.011106176301836967, 0.4779239594936371, -0.33538755774497986, 1.9620347023010254, 2.285916805267334, 0.8905056118965149, 2.6433351039886475, 1.0223119258880615, 2.5713493824005127, 0.34323740005493164, 0.07853814214468002, 0.9205058217048645, 0.3103882074356079, -0.5093503594398499, -0.2127789407968521, 0.4627962112426758, -0.27298423647880554, -1.1098518371582031, -2.4940712451934814, -0.40426766872406006, 0.019428325816988945, -1.5258729457855225, -0.45156583189964294, -0.44478315114974976, -0.6142958998680115, -0.34805309772491455, -1.8537030220031738, 0.46561092138290405, 0.4484243392944336, 2.8421621322631836, -0.20608237385749817, 0.5738343000411987, 0.35120201110839844, 0.19046920537948608, 0.14453016221523285, 0.3563102185726166, -0.5124994516372681, -1.6389272212982178, -0.4052180051803589, 0.014407186768949032, -0.4227001965045929, -0.15023325383663177, -0.8172345757484436, -0.9002220630645752, -0.4789639115333557, 0.0226602740585804, -0.3830203711986542, -0.9058100581169128, 1.0814036130905151, 0.06429638713598251, 0.006550712510943413, -0.7132749557495117, -0.7607928514480591, 0.9298311471939087, 0.0943666398525238, 0.8215736746788025, -2.3039283752441406, -0.40513497591018677, 0.9448987245559692, 0.21605485677719116, -0.7402067184448242, -0.2434270828962326, -0.044134583324193954, 0.2005649358034134, 0.2589613199234009, -0.8996691107749939, 0.2846692204475403, 0.3008158206939697, 0.9488381147384644, 0.29108503460884094, -1.4642879962921143, 1.4348828792572021, -0.9381688237190247, -0.6443460583686829, -0.6525417566299438, 0.6677617430686951, 1.1440858840942383, -0.03949694707989693, -0.5593855381011963, 0.34907621145248413, -0.7498024106025696, 0.7507708668708801, 0.4503003656864166, 0.6429666876792908, 0.6284415125846863, -0.2903331518173218, 1.0075654983520508, 0.996036946773529, 0.8897716999053955, -0.025820352137088776, -0.5304960608482361, -0.143401637673378, -0.1327877640724182, -2.049820899963379, 1.0958032608032227, 0.1359494924545288, 0.03624375909566879, -0.2701130211353302, -0.17305153608322144, 0.8576912879943848, -0.6259546279907227, 0.24852660298347473, 0.03207516297698021, 0.1888962984085083, 0.7517452836036682, 0.010536223649978638, -0.7119218111038208, 1.055364966392517, 2.219409227371216, -1.1816822290420532, -1.82333242893219, 0.4810689687728882, 0.4260157346725464, -0.4979804456233978, 1.0554490089416504, 0.3972700536251068, -1.629647135734558, -2.1823196411132812, 0.01888165809214115, 1.4006543159484863, -1.2008695602416992, 1.709304690361023, 0.3995072841644287, 0.15071401000022888, 0.7275082468986511, -1.3654884099960327, -0.8102589249610901, -0.728213906288147, -1.7333353757858276, -0.8023285865783691, -2.101116418838501, 1.0229629278182983, -0.4143050014972687, -0.18936768174171448, 0.2116815447807312, -0.5467135310173035, -0.5291318297386169, 1.138063669204712, 0.9556791186332703, -1.4086860418319702, 0.8984512686729431, 1.3270448446273804, -1.177112340927124, 0.6284141540527344, -1.0957694053649902, 0.194500133395195, -0.3800427317619324, 0.592903733253479, 1.155776023864746, -0.5698330402374268, 0.8047451972961426, -0.0050069247372448444, 0.7130500674247742, 0.3929824233055115, -0.3175608515739441, -0.5587764978408813, 0.23447583615779877, 1.7177766561508179, -0.41395965218544006, 0.805882453918457, 1.9281799793243408, 1.260275959968567, -0.5886760950088501, -1.2337782382965088, -0.8067518472671509, -0.9649004340171814, -0.37735220789909363, 0.2218172252178192, 1.3559492826461792, 0.8610746264457703, -0.363662451505661, 0.039764050394296646, 0.46878179907798767, -0.43844878673553467, 1.2614624500274658, 0.11270667612552643, 1.4053289890289307, -1.6590057611465454, 0.5899112224578857, -0.019318656995892525, 0.270289808511734, -0.10150686651468277, 0.6175954937934875, 0.18096297979354858, -0.5374600887298584, -0.6908532977104187, -0.09835165739059448, 0.9204962849617004, 1.1905744075775146, 0.2573780417442322, -0.5712411403656006, 0.8516814112663269, 0.42651501297950745, 1.211159348487854, -0.5543016195297241, 0.8209282159805298, -0.8502349853515625, -0.578855574131012, 0.4071780741214752, 0.47359609603881836, 0.5528525710105896, 0.09986478835344315, 2.302030563354492, -0.12652479112148285, 0.6542834639549255, -0.28994452953338623, 0.18950627744197845, 1.7486441135406494, 0.17332154512405396, 0.08585350960493088, -1.7177271842956543, 0.37763622403144836, 0.4511561393737793, -0.4267564117908478, 0.3092830181121826, -0.2683894634246826, -0.43211764097213745, 0.7030728459358215, 1.013808012008667, 2.877192974090576, -0.9525319337844849, -0.2694481909275055, -1.2717994451522827, -0.38687530159950256, -0.6260366439819336, -0.49261677265167236, -0.27364328503608704, 0.2639189064502716, -1.230103850364685, -1.2862677574157715, 1.1561098098754883, 0.3854106664657593, 0.26505807042121887, 0.7841082215309143, -0.2309104949235916, -0.604409396648407, -1.4417996406555176, 0.9429270625114441, -1.0768373012542725, -0.06504455953836441, -0.25894173979759216, -0.05547985062003136, -0.41578298807144165, 0.3841625452041626, 0.9170160889625549, -0.459037721157074, -1.232645034790039, -0.7334514260292053, 0.32377997040748596, 1.267546534538269, -0.3947267532348633, 0.7822417616844177, 1.7897714376449585, -1.5197972059249878, 0.16337676346302032, -0.10329718887805939, -2.3209023475646973, 0.9940934181213379, -0.11192820966243744, -0.18830017745494843, -0.030296094715595245, 0.7458913922309875, -0.09751464426517487, -0.7446109056472778, -0.2979030907154083, 0.25480231642723083, 0.8344922065734863, 0.6963713765144348, 0.28422704339027405, -1.0360227823257446, -0.38531506061553955, 1.594527244567871, 0.8842783570289612, -0.3577042818069458, -2.2683725357055664, 1.1250702142715454, 0.4739472270011902, 0.6045615673065186, 0.5569500923156738, 0.3969925343990326, -2.0744879245758057, 0.2814372479915619, -1.0470607280731201, -0.2575887441635132, 1.2700804471969604, 1.0574982166290283, 0.14415138959884644, 1.3446778059005737, -1.779187560081482, 1.4180752038955688, -2.080489158630371, 0.2102409452199936, 1.4443120956420898, -0.19309377670288086, -0.8243380188941956, -0.009898209944367409, 0.9900888800621033, -0.0018834039801731706, -1.5614784955978394, 0.3604365587234497, -0.36890095472335815, -1.2426233291625977, -0.08450658619403839, -1.1297141313552856, -0.6131344437599182, -1.8377084732055664, 0.3656109571456909, 0.759864330291748, 0.8358413577079773, -0.5265738368034363, -1.5422435998916626, 1.0304275751113892, 2.568657636642456, -1.4250586032867432, 2.3126001358032227, -0.27822956442832947, 0.04820633307099342, 1.5674246549606323, -0.7239087224006653, 0.25331035256385803, 1.2749207019805908, 2.215325355529785, 2.0085291862487793, 0.735153317451477, 0.012455048970878124, -2.123840093612671, 0.9838046431541443, 0.4029994010925293, -0.04977430775761604, 2.3443570137023926, 1.1020097732543945, 0.48498642444610596, -0.7044079899787903, -0.38979408144950867, -0.5928568243980408, 0.673328697681427, -0.31218692660331726, -0.14940866827964783, -0.6288859248161316, 0.8069409132003784, 0.5711342692375183, 1.5454468727111816, 1.8046133518218994, -0.7217859625816345, -0.6139814853668213, -0.7774226665496826, 1.5323866605758667, -0.9368578195571899, 0.33367252349853516, 0.3335208594799042, 0.03208035230636597, 0.5242831707000732, -0.07583148032426834, -0.59298175573349, 0.27048563957214355, -0.814710259437561, -1.425223708152771, -1.185275673866272, 0.535026490688324, -2.0241215229034424, 1.3174653053283691, -1.1848080158233643, 1.147802472114563, -0.7416799068450928, -0.5256150364875793, 1.590671420097351, -0.8445256948471069, -0.254082053899765, 1.9432560205459595, 0.2811664938926697, 1.13955557346344, -2.4439547061920166, 0.9986122250556946, 1.295306921005249, 0.9386895895004272, -0.2619146704673767, -0.1057538241147995, -1.299279808998108, 0.3780844807624817, -0.9193588495254517, 0.6431930065155029, 0.791543185710907, 1.1319117546081543, 2.496412754058838, 1.3742334842681885, -1.6024799346923828, -0.5766046047210693, 1.1279603242874146, -1.1413366794586182, -0.91112220287323, 0.36337777972221375, 1.2200263738632202, -0.16221188008785248, -0.4675110876560211, 1.59396231174469, -1.7481694221496582, -0.2661984860897064, 0.4547634422779083, -0.6876352429389954, 0.5175474286079407, -0.643427848815918, -0.07939784973859787, -1.146905541419983, -0.7251653671264648, -1.0731351375579834, -0.9809821248054504, -0.9558744430541992, -0.02252657525241375, 1.5170069932937622, 0.334929883480072, -0.9686455130577087, 1.0099828243255615, -0.07164006680250168, -1.318027138710022, 0.6548104286193848, 0.8310638070106506, 0.8355337381362915, 0.49574100971221924, 1.430618405342102, 0.10834864526987076, 1.5264089107513428, 0.14109265804290771, -0.5365949273109436, 1.170586109161377, -0.8942368626594543, 0.5457375049591064, -1.678851842880249, 0.29076170921325684, 0.7534993290901184, -0.16410405933856964, 1.8133692741394043, 0.7041539549827576, -1.6525771617889404, 0.8010486960411072, -1.5302835702896118, 1.1733487844467163, 0.02191188372671604, -1.8205974102020264, 1.6125235557556152, -0.12684683501720428, 1.105507493019104, -0.7632038593292236, -1.658219575881958, 0.5175105929374695, -1.111720085144043, -0.4866922199726105, 0.13481566309928894, 0.7668150067329407, 1.0278841257095337, -1.9938896894454956, 1.4990932941436768, -0.001774731557816267, -0.2485569268465042, 2.2954928874969482, 0.8353431820869446, 1.2969136238098145, 0.3891496956348419, 1.1160064935684204, 0.1413397490978241, 1.1487131118774414, 0.16988740861415863, -1.2487133741378784, -0.42879724502563477, -1.2217775583267212, -1.2076261043548584, -1.267520785331726, 0.6024333238601685, 1.2986812591552734, 0.5748283863067627, 1.3753069639205933, 0.4334632158279419, -2.014683485031128, -0.4719979465007782, 0.12393214553594589, -0.6124032139778137, 0.9538440704345703, -0.10219930112361908, 0.5932435393333435, -1.2578028440475464, 1.0883854627609253, -0.5196503400802612, -0.20923294126987457, -0.6064682006835938, -1.2072556018829346, 0.1633284091949463, -0.6420322060585022, -0.26878148317337036, 0.36757606267929077, -0.08412431180477142, 0.07845644652843475, 0.7245195508003235, 0.6833769083023071, -0.2641419768333435, -0.3902941048145294, 1.3760950565338135, -0.2639613449573517, -0.5179153084754944, 0.7337671518325806, 1.8119006156921387, 0.021622734144330025, 1.7546106576919556, -0.6974920630455017, -0.7949033975601196, 0.7822118401527405, -0.22954599559307098, 0.14879247546195984, 0.19545193016529083, 0.5694425702095032, 0.5413908958435059, -0.5255693197250366, -0.06160372495651245, -1.439771056175232, -0.9292914867401123, 1.2314282655715942, -0.44999849796295166, -1.1903917789459229, -1.3276569843292236, -0.45224231481552124, 1.237640619277954, 0.3683421015739441, 0.3612162470817566, -0.25277432799339294, 0.5768750905990601, -0.029559694230556488, -0.9778994917869568, -1.7629672288894653, 0.27540677785873413, -0.9916161298751831, 0.8351172804832458, -1.4897661209106445, -0.2643427550792694, 0.21101757884025574, 1.6931493282318115, -1.0672746896743774, -0.5440661311149597, -0.1414102464914322, -0.40676796436309814, -1.125488042831421, 1.9390853643417358, 0.71487957239151, 1.6461338996887207, 2.0245883464813232, 1.155296802520752, 1.047418475151062, 0.19760315120220184, -0.3195610046386719, -1.1165510416030884, -1.7315558195114136, 0.5133125185966492, 1.3448106050491333, -2.023674964904785, 1.6752222776412964, -1.248756766319275, -0.8949147462844849, -1.3549742698669434, 0.119442418217659, 0.8188827633857727, 0.03640449047088623, 1.6794114112854004, 0.5889316201210022, -0.6014199256896973, 1.2183866500854492, 0.9389408230781555, -0.3271113336086273, -0.6059066653251648, 0.5117213726043701, 0.42568618059158325, -0.299375057220459, 0.5979200601577759, 1.6480889320373535, -0.48499760031700134, 0.8487957715988159, -0.18534301221370697, -0.22427284717559814, -0.6898714900016785, -1.0417011976242065, 0.7760084867477417, -0.8062177300453186, 0.024410368874669075, 1.5005061626434326, -2.0463438034057617, -0.05785754323005676, 0.2639402151107788, 0.31462496519088745, -1.3204543590545654, 1.5753107070922852, -0.9965674877166748, 0.1234714537858963, -0.015844151377677917, 0.43479374051094055, -1.5147662162780762, -1.4822410345077515, 0.25048214197158813, -0.06728994846343994, 0.8721770644187927, -0.4387785792350769, 2.2014849185943604, -0.6914160847663879, 0.744903564453125, 1.6909540891647339, -0.3994814455509186, 0.8299999237060547, 0.3812473714351654, -0.3857824206352234, -1.0216628313064575, -0.11340843141078949, 1.1869317293167114, 0.9611743092536926, 0.48308324813842773, 0.5406250357627869, -0.7076928019523621, 0.3212260603904724, -0.557009220123291, 0.7158411145210266, -0.5525191426277161, -0.7269703149795532, 0.8254331946372986, 0.8539828658103943, -0.6593995094299316, 0.35366013646125793, 0.16822928190231323, -0.5963497161865234, 1.4703655242919922, 1.521022081375122, -0.5286790132522583, -0.9298908710479736, -1.4546630382537842, -0.6309178471565247, -1.0214835405349731, 0.926744282245636, -0.5006264448165894, -2.0308289527893066, 0.5337246060371399, 1.2118300199508667, 0.4024132490158081, 1.0763640403747559, 1.4807026386260986, 0.6584159731864929, 0.7792165875434875, 1.6361048221588135, -0.5966443419456482, -0.8504359126091003, -0.1265985667705536, -1.6873568296432495, 0.07915201038122177, 0.11570890992879868, 0.30891984701156616, 0.029756292700767517, -0.9471812844276428, 0.6787979006767273, 0.8235626220703125, -1.5570141077041626, 0.6921533942222595, -0.7921214699745178, 0.09484193474054337, 1.1811834573745728, 1.6750212907791138, 1.6286251544952393, -0.8876007199287415, -0.3042196035385132, -0.5357558727264404, -1.503868579864502, -0.18852809071540833, -0.885807991027832, -0.9538531303405762, -0.18504202365875244, -2.004119873046875, -0.2906545102596283, -0.6743436455726624, -2.1684858798980713, -0.4777017831802368, 0.7057723999023438, 2.281372308731079, 1.3570090532302856, -0.22454623878002167, -1.0333664417266846, 1.2121307849884033, 0.6605041027069092, 0.6316668391227722, 0.2900877594947815, 0.08744348585605621, -0.696100115776062, 2.456313371658325, 2.530945062637329, 1.4527968168258667, -0.2204757034778595, 0.6164919137954712, -2.069735050201416, -0.5461077094078064, -0.9702299237251282, -0.06861782819032669, -0.27060338854789734, -0.018806176260113716, 0.3852996826171875, -0.11034756153821945, 0.4781181216239929, 0.42367053031921387, -0.5275819897651672, 0.16658338904380798, -1.5026599168777466, 1.4806244373321533, -0.13744454085826874, -1.6878323554992676, -0.3328966498374939, -0.2499236762523651, 0.9331438541412354, -1.448567271232605, 0.026790104806423187, 0.6443144679069519, -0.6760187745094299, -0.0269540473818779, 0.22541698813438416, -0.08922216296195984, -0.8805606365203857, -0.4232384264469147, -0.15784426033496857, -1.2034339904785156, -0.9578679800033569, 1.6728196144104004, 1.2206032276153564, -0.00926921982318163, -0.032114092260599136, -0.9843196272850037, 1.1473441123962402, 0.6268665194511414, -0.8882394433021545, 1.66580069065094, 0.8314234018325806, 0.7107171416282654, 0.43391767144203186, -0.1809297353029251, -0.18478363752365112, -0.9546067714691162, 1.8776806592941284, -0.6737693548202515, 1.5780256986618042, 0.27058473229408264, 1.0856401920318604, 1.0989574193954468, 0.5986669659614563, 1.543355107307434, 0.7805989980697632, 1.018467664718628, 0.43476611375808716, 0.3274425268173218, -0.06305036693811417, -0.2152106910943985, 0.6134507060050964, -0.9386391639709473, 0.6401903629302979, 1.4182748794555664, -0.9973816871643066, -1.7783434391021729, 0.26176804304122925, 0.9028847813606262, -0.461174339056015, 1.4220672845840454, -0.47654783725738525, -1.5395525693893433, 0.1456986963748932, 0.4490715563297272, -1.1357396841049194, -1.9149247407913208, -1.041062831878662, 1.3894193172454834, 1.6801356077194214, 0.8174170851707458, 0.2364167720079422, 1.701256513595581, -0.7894731760025024, -1.5581550598144531, 0.036806534975767136, 0.5231810212135315, 0.22485260665416718, 0.2580476701259613, 1.8422318696975708, -0.021351944655179977, 1.0457628965377808, 0.38007065653800964, 0.8494729995727539, -0.12693500518798828, -0.5473929643630981, 0.5031039118766785, -0.05201995000243187, 0.3149849474430084, -0.7338293194770813, 0.9666099548339844, 0.0436411090195179, 0.258513480424881, -1.1787705421447754, -0.6277408003807068, 2.0117502212524414, 2.125406503677368, -1.0516865253448486, 0.1603352278470993, -0.7556295394897461, 0.7239117622375488, 0.9876301288604736, 0.5675589442253113, 0.6502144932746887, 1.1316592693328857, 1.4851783514022827, 0.734630286693573, 0.1362139880657196, -1.5831880569458008, 1.370811939239502, -0.3780301511287689, 0.6526215076446533, 0.3584701716899872, 0.4735572338104248, 0.6234757900238037, -0.7340546250343323, 0.6835001111030579, 0.33102700114250183, -0.36919111013412476, 0.3759864270687103, -0.3323141932487488, -0.04825691878795624, -0.6873050332069397, -0.7063486576080322, -1.28868567943573, 0.9444831013679504, 0.12684085965156555, 1.1265000104904175, -0.17986081540584564, 0.3141421973705292, 2.355184555053711, 0.4383777379989624, -1.0843955278396606, 0.6272676587104797, 0.1325107365846634, -1.2094523906707764, -1.9275081157684326, 1.1582192182540894, 1.5750967264175415, -1.3394414186477661, 1.4814121723175049, 0.4972924292087555, 1.5560829639434814, -3.0810904502868652, 2.584043264389038, 1.9099633693695068, 0.08882803469896317, 0.4154588282108307, 0.6429044604301453, 1.3784734010696411, 0.8302733898162842, 0.17562097311019897, 0.2603227496147156, 0.17107000946998596, -0.01315249502658844, 1.008156418800354, 1.2048826217651367, -0.7471878528594971, -0.855773389339447, 2.698791980743408, 0.04147918149828911, -0.5761659145355225, -2.0478363037109375, 1.4107667207717896, -0.8336660265922546, -1.6311033964157104, -0.82956862449646, -0.04856749251484871, -1.3272813558578491, -0.5380903482437134, 1.9179514646530151, -0.6469959020614624, 0.3259985148906708, -1.030310869216919, 1.8328033685684204, 0.12437961995601654, -0.1889984905719757, -1.4661462306976318, 0.9306204915046692, 0.8103839159011841, 0.4901496469974518, 1.6571489572525024, 1.063673973083496, -0.46205615997314453, -0.7718790173530579, 0.3390766680240631, 0.019122451543807983, 0.319227010011673, -1.025421142578125, 0.13015660643577576, 0.6105398535728455, -0.236162468791008, -0.34691599011421204, 1.4626909494400024, -1.7314386367797852, -0.4904429614543915, 0.08473774045705795, 1.0125572681427002, -0.07278299331665039, -0.24384500086307526, -0.04961176961660385, -0.34391286969184875, 0.8772834539413452, 0.2629087269306183, 0.06382538378238678, 1.4950441122055054, 1.0980304479599, 0.25676229596138, -1.752077579498291, -3.0006306171417236, 0.09513666480779648, -0.1187305897474289, 2.4243123531341553, -0.2815774977207184, -0.8904399275779724, -0.31414222717285156, -0.32330095767974854, -1.4638280868530273, 1.8250535726547241, 0.1949865072965622, 0.7353823184967041, -0.44383329153060913, 1.6847108602523804, -1.0410144329071045, 0.7812073826789856, 0.8949547410011292, -0.4840323030948639, -0.722511887550354, -0.20997591316699982, 0.4719107449054718, 0.1565978229045868, 0.656548798084259, 0.9518646597862244, 1.3793492317199707, 0.762525737285614, -0.5663597583770752, 0.009290437214076519, 0.2508786916732788, 2.223356008529663, -1.008740782737732, 1.7270479202270508, -0.4364997148513794, 1.3041318655014038, -1.9815373420715332, 0.31679970026016235, -1.6431266069412231, 1.2353979349136353, 0.18517208099365234, 1.13252592086792, 0.7135061025619507, -2.200751781463623, -0.4586229622364044, 0.20866671204566956, -0.8782302141189575, 0.7487035989761353, -0.3010333776473999, -1.2588353157043457, 0.7612356543540955, -2.2138590812683105, -0.43999505043029785, -0.9264640212059021, 1.8644671440124512, -1.1378653049468994, -0.26794207096099854, -0.37610751390457153, 1.6948400735855103, 0.4707362651824951, -0.14171595871448517, 0.7672625780105591, -1.3007490634918213, -0.15895149111747742, 0.11818026006221771, 0.19614426791667938, -0.5068166851997375, 0.9582255482673645, -0.42914852499961853, 0.9387273192405701, -1.788565754890442, 0.4511553645133972, -1.272091269493103, 1.5029205083847046, -1.3679405450820923, 0.30029022693634033, 0.47448599338531494, 0.6149166226387024, 1.162666916847229, 0.7452923059463501, -0.029844924807548523, 0.38410890102386475, 0.4545024633407593, -0.44128161668777466, -0.7385048270225525, 0.6953263878822327, 1.68578040599823, 1.691052794456482, 0.2053641825914383, -0.9770359992980957, 0.5269737243652344, 0.8258283734321594, 1.1606240272521973, -1.062540888786316, -1.6430809497833252, 1.9000749588012695, -0.8708942532539368, -0.10842897742986679, -0.10460027307271957, 1.7309175729751587, -1.017375111579895, -0.7745881080627441, -0.5952349305152893, 2.770886182785034, 1.4952210187911987, -1.5763458013534546, -1.4476871490478516, 0.16212819516658783, 2.2152602672576904, -0.7081558108329773, 1.4307724237442017, 0.5249650478363037, 0.09681448340415955, 0.711682140827179, -0.5123084783554077, -0.44600415229797363, 0.62109375, 0.849524974822998, 0.8516800403594971, 0.3577534556388855, 0.8325110077857971, -0.6514227986335754, -0.23028399050235748, -0.4160500466823578, 0.05133017525076866, 0.2738202214241028, -0.08869641274213791, 0.22809992730617523, -3.075444459915161, 0.11406145244836807, 0.5939792990684509, 0.38583454489707947, 0.1527300924062729, 1.1931788921356201, -0.029704058542847633, -1.6720941066741943, 0.9788303375244141, -0.5406467914581299, -1.1626392602920532, -0.2835720479488373, 0.12144435942173004, -0.8622706532478333, -1.0461350679397583, -0.8657816648483276, 1.6531857252120972, 0.3025226294994354, 0.3451492190361023, -0.3064805865287781, -0.012761968187987804, 0.5270192623138428, -0.5596306920051575, 0.5809727907180786, 1.526734471321106, 0.2184862643480301, 0.8469110131263733, 0.10607681423425674, -0.2328099012374878, 0.46404367685317993, -1.275068998336792, 0.6714633703231812, 2.374025821685791, -0.5303343534469604, 1.0873509645462036, 0.21064874529838562, 0.7417325973510742, 0.08059047162532806, 0.23189115524291992, 0.7212160229682922, 0.4966756999492645, 0.39135804772377014, -0.31390270590782166, 1.7883399724960327, -0.692646861076355, 1.1416250467300415, -0.7797420620918274, 1.081850528717041, -0.6646647453308105, -0.03517685458064079, 1.899821162223816, 0.39531177282333374, -1.0178605318069458, 0.3236209750175476, -0.2730101943016052, 0.3402164578437805, 0.8375414609909058, 0.6794258952140808, 0.6438579559326172, 0.9593054056167603, 0.35343390703201294, -1.1506954431533813, 1.2555418014526367, 1.2560850381851196, 0.4273683428764343, 0.5691816210746765, 1.0724612474441528, 0.07974656671285629, 0.26580333709716797, -0.05536632612347603, -0.2727425992488861, -0.7717739343643188, -0.23756442964076996, 0.1771257072687149, -0.7017691731452942, -0.1362573355436325, -1.8170794248580933, -1.137247920036316, -0.25812238454818726, -1.5731539726257324, -0.6732553839683533, -1.1904373168945312, 1.3575043678283691, 1.9431763887405396, -0.8779888153076172, -1.012035846710205, -0.695172131061554, -0.3040485084056854, -0.14398254454135895, 1.401990294456482, -0.651146650314331, -1.0505363941192627, 1.605212688446045, -0.348583847284317, 1.1530042886734009, -1.0985243320465088, -0.3627275228500366, 1.3165417909622192, 0.5404231548309326, 2.0582103729248047, 1.667602300643921, 2.448016881942749, -0.697242021560669, 2.0769131183624268, -0.6131758093833923, 0.3302103877067566, -0.11545191705226898, 1.029006838798523, 0.8796459436416626, -0.8408886790275574, -0.2358844131231308, 0.3361028730869293, 0.18745020031929016, -0.2547977864742279, -0.25542888045310974, -0.9664531946182251, -0.5276089906692505, 0.8781973123550415, -0.22350692749023438, 0.16002491116523743, 0.032798465341329575, 0.6200730204582214, 0.39049869775772095, -0.09028401970863342, -0.5036953091621399, -0.47215452790260315, 1.316108226776123, -1.1451975107192993, 0.8275641798973083, 0.33834537863731384, 0.39547520875930786, 2.018242835998535, 1.5691150426864624, 1.6237308979034424, 1.8851743936538696, -0.3156762719154358, -1.022135615348816, 1.3599625825881958, -0.5193729996681213, -0.8295860886573792, -1.8469160795211792, 0.5189220309257507, 1.434939980506897, 0.32766103744506836, 0.4204927682876587, -0.23840032517910004, 1.252540946006775, 0.052163247019052505, 0.1080465018749237, 0.12465247511863708, -0.3885294795036316, 0.3608885407447815, -0.7708569169044495, -1.0680264234542847, 0.01906444877386093, 0.4504009187221527, 2.479705333709717, 0.6739874482154846, 0.37236884236335754, -0.9392135739326477, -1.5691567659378052, 0.7249312996864319, 0.9584572911262512, -0.21550890803337097, -0.6710473299026489, 1.0054641962051392, 0.9738855361938477, 1.2032122611999512, 1.4830948114395142, 0.5056135654449463, -0.1789853274822235, 0.5439825654029846, -0.2156120389699936, -1.4645872116088867, -0.45856064558029175, -0.8399760127067566, 1.0893179178237915, 0.24197480082511902, -0.5546174645423889, 0.8922102451324463, -0.5814225673675537, 0.31820932030677795, -1.464890956878662, -0.5652115345001221, 0.7543608546257019, -0.5843625664710999, 0.31888219714164734, -0.5496626496315002, -0.6186855435371399, 0.09112212806940079, 2.3680851459503174, 0.8286955952644348, -0.2948518693447113, 0.08884385973215103, 0.9962268471717834, -0.4923454523086548, -0.08799519389867783, -1.236746907234192, -1.8493120670318604, 0.3319043517112732, 0.9479086995124817, -0.08856549859046936, -0.28333330154418945, 0.1555456668138504, -0.5158466696739197, -0.6804214715957642, 0.11011018604040146, 1.5371720790863037, 0.23896996676921844, 1.5448423624038696, -0.44925183057785034, 0.7766723036766052, -0.7970913052558899, 0.560692548751831, 0.7900442481040955, 1.0569984912872314, 0.2292865365743637, 0.707859218120575, 0.1413046419620514, 1.3683819770812988, -0.8805944323539734, 0.824860692024231, -1.1134263277053833, -0.35440683364868164, 0.797663688659668, -1.5612375736236572, -0.13808389008045197, -0.7791944146156311, 0.3134097158908844, -0.2759634554386139, -0.4480862617492676, -0.13460491597652435, -1.1810156106948853, -1.234075903892517, 0.2566705048084259, -1.1854147911071777, 1.4513416290283203, 0.2515130341053009, 1.1111036539077759, -1.0913394689559937, 0.1511046141386032, 0.007685324177145958, 0.1340264528989792, -0.377841055393219, 0.28774532675743103, -0.17069223523139954, 0.49756544828414917, -0.5442430377006531, 0.4783836305141449, 0.5341041088104248, 2.0805485248565674, -1.1029086112976074, -0.13821856677532196, 1.4415984153747559, -0.4832358956336975, 0.08280142396688461, 0.9120625853538513, -1.4524874687194824, -2.1967010498046875, -0.23945067822933197, -0.6056600213050842, -1.905123233795166, -0.14768001437187195, 0.014280805364251137, 0.12511374056339264, 1.7586994171142578, -0.5303347110748291, -2.3084936141967773, -1.8697997331619263, -0.2683122158050537, 0.16064366698265076, 0.5990349650382996, 1.3046444654464722, -1.4441500902175903, -0.13983389735221863, -0.21373245120048523, 1.5956887006759644, -1.2251619100570679, -0.5535997152328491, -0.38269054889678955, 1.7106702327728271, 0.5730488896369934, 1.1550867557525635, 0.2067250907421112, 0.9939852356910706, -0.6260943412780762, 0.5123827457427979, -0.8280360102653503, 0.9191922545433044, -1.1565077304840088, 0.3857021927833557, -0.23615410923957825, 1.7263039350509644, -1.4426019191741943, 1.153434157371521, -0.8025432825088501, -1.5459003448486328, -0.5654227137565613, -1.0683398246765137, 0.030912885442376137, -0.4287724792957306, 1.3086990118026733, 1.1444119215011597, 0.1952926069498062, 0.32299113273620605, 1.3371554613113403, 1.6731343269348145, 1.5464307069778442, -0.2640671133995056, -0.9052299857139587, -1.02507483959198, 0.9911677837371826, -0.003668292658403516, -2.6849799156188965, -1.1154643297195435, 1.0407638549804688, 0.7626004219055176, 0.19440573453903198, -1.9751545190811157, 0.5084770917892456, 0.819830060005188, -1.8169152736663818, 0.45484259724617004, -0.4650592803955078, -1.2896779775619507, -0.04986754432320595, 0.4209609627723694, -0.42722487449645996, 0.8513679504394531, 0.19137687981128693, 0.7327555418014526, -0.8479138612747192, 0.1811426430940628, -2.076977491378784, 0.09047539532184601, -1.0571315288543701, 0.157410129904747, -0.9831701517105103, -0.6506217122077942, -0.022123724222183228, -0.0665469691157341, 1.0472321510314941, -1.7904019355773926, 1.8097513914108276, -1.2110620737075806, -0.8090900182723999, -0.3636154234409332, -1.8346437215805054, -0.42388880252838135, 1.4702948331832886, -2.100492477416992, 0.1643359363079071, -0.017850792035460472, -1.3905164003372192, 0.06652773171663284, 0.4161324203014374, 0.7701910734176636, -1.1962023973464966, 1.1726970672607422, -0.6636601090431213, 0.18912425637245178, -0.491561621427536, -0.1803637146949768, -0.6614829301834106, 0.4329913556575775, -0.13759444653987885, -0.37636205554008484, -1.0546990633010864, 1.297922968864441, -1.9344736337661743, 1.654112458229065, 1.4862298965454102, 0.037287767976522446, 0.7757458090782166, -0.8428913354873657, -1.2807525396347046, -1.0205857753753662, 0.3404412567615509, -1.1157574653625488, 2.5377094745635986, -0.917437732219696, 0.33467206358909607, 0.9547212719917297, -0.8529514670372009, -0.46146759390830994, 0.019060641527175903, -0.9000529646873474, -0.7978154420852661, 0.12695030868053436, -0.4416438639163971, -0.3397582769393921, 0.8734787702560425, -1.9663530588150024, 1.5686739683151245, 0.17099575698375702, 0.0006124629871919751, -0.03950214758515358, 0.8121763467788696, 2.466104507446289, 1.903739333152771, 0.5369753241539001, 1.7278128862380981, -1.3497514724731445, 0.31777554750442505, -1.5451579093933105, -0.8085846304893494, -0.1694760024547577, -1.422273874282837, 1.078979253768921, -0.271732896566391, 0.6839122176170349, 0.417309045791626, 1.2484517097473145, -0.8935521841049194, 0.8754113912582397, 0.034538693726062775, -0.19058658182621002, 0.5009121894836426, -0.4978564381599426, 2.144530773162842, 0.13332562148571014, -1.97786283493042, -0.466497004032135, -0.31810542941093445, -1.065845012664795, 0.4860880970954895, 0.7281827926635742, 0.4587048888206482, -0.5520063042640686, 0.12801425158977509, 0.0022169228177517653, 2.036302328109741, -1.8898570537567139, -2.0046262741088867, -0.012511125765740871, 0.890437126159668, -0.9998286962509155, 0.20260852575302124, -0.013025506399571896, 0.4612785279750824, -1.507459044456482, 0.668430745601654, -0.25425100326538086, -1.1672996282577515, 2.302445650100708, -0.9142683744430542, 1.3481221199035645, -1.136684775352478, 0.22073791921138763, -1.5847712755203247, 0.005469832569360733, 0.652993381023407, 0.9127197265625, 0.5498961210250854, -1.719600796699524, -0.5269829630851746, -0.14709848165512085, 0.9959413409233093, -1.2420669794082642, 0.30269280076026917, 0.5572394132614136, 0.7242938280105591, 0.5322431325912476, -0.5840322971343994, 0.6445144414901733, 0.959792971611023, -0.4184030294418335, -1.7794177532196045, -1.0521496534347534, 0.5827885866165161, -0.706713080406189, 0.9484006762504578, -0.24563723802566528, -0.4053240418434143, 0.9686067700386047, 0.7174842357635498, 0.5623261332511902, 0.9318108558654785, -0.9501004219055176, 0.07473434507846832, -0.05266176164150238, -0.422720730304718, -0.24866066873073578, -0.6290907263755798, -0.0023605898022651672, -0.5059103965759277, 0.9844160676002502, 0.5301871299743652, 1.457819938659668, 1.469934105873108, 1.1056914329528809, -1.0547726154327393, -0.3334733545780182, 0.2027832567691803, -1.5520275831222534, 0.14774483442306519, 0.08943655341863632, 0.15260480344295502, 1.4917956590652466, -0.48397383093833923, -0.01364942081272602, -0.44405943155288696, -0.27506116032600403, -0.761528491973877, -1.033661961555481, 1.302988052368164, -1.6228182315826416, 0.13424867391586304, 0.3537742495536804, 0.6669729948043823, -1.427879810333252, 0.5124143958091736, 0.2942076623439789, -1.097038745880127, -0.113441601395607, -1.2431432008743286, 0.3605928122997284, -0.2811623215675354, -0.43765828013420105, 0.6247261166572571, -1.05177903175354, 3.9071269035339355, 1.1289397478103638, 0.463570773601532, 0.6467713117599487, 0.12116852402687073, -2.5568742752075195, 0.4142494797706604], [-0.7941503524780273, 0.48755592107772827, -1.6051288843154907, 1.055243968963623, 0.9662265777587891, 1.2598408460617065, -0.25158512592315674, -2.455848217010498, -0.5899257659912109, -0.02095806412398815, 0.12969906628131866, -1.74277663230896, -0.327538400888443, 1.3558080196380615, -0.6581282615661621, -0.3223303258419037, -0.8404099345207214, 1.040586233139038, -0.1696990728378296, 0.6686056852340698, -0.8060554265975952, 0.21477892994880676, 0.30704209208488464, 0.5642387270927429, -0.39448612928390503, 0.8520565629005432, 0.47451817989349365, -1.3204929828643799, 0.14215700328350067, -0.03621334210038185, 0.7477242350578308, -0.292650043964386, -1.5858817100524902, -0.547629177570343, -0.047283582389354706, 0.10074649751186371, -0.45314353704452515, -0.9712076783180237, 1.2909640073776245, -1.0736610889434814, -0.5112877488136292, 0.39558184146881104, -1.2795857191085815, -0.9706450700759888, -1.4959501028060913, 1.0740493535995483, -0.16720551252365112, -0.5806517601013184, -1.0376298427581787, 0.5732991695404053, 1.3395061492919922, 0.7133283019065857, -0.23227761685848236, 1.1718343496322632, -1.1368948221206665, -2.043534755706787, -1.298667550086975, 0.5580444931983948, 1.4716277122497559, 1.130537986755371, -1.0794737339019775, -0.9142538905143738, 0.6490485072135925, -1.1066786050796509, -1.206666111946106, 0.8931307196617126, -2.961843967437744, 0.6558861136436462, -0.37205958366394043, -0.22532841563224792, 0.38155385851860046, 0.7080431580543518, -1.3216798305511475, -0.5679197311401367, 0.778211772441864, 0.9082036018371582, -0.06552417576313019, -0.19094395637512207, -0.01250722911208868, -1.1758157014846802, 0.9904309511184692, 0.5606228709220886, -1.1328202486038208, 0.9044535160064697, -0.16139866411685944, 0.695773184299469, -1.2880617380142212, -0.015522447414696217, -1.0589762926101685, -1.4629771709442139, 0.4068135619163513, 0.40597110986709595, -0.717807948589325, -0.04503412917256355, 1.2157379388809204, -1.067949652671814, 0.520736575126648, 0.5719025731086731, 0.4328552186489105, -0.8741248250007629, 1.7912105321884155, 0.6075193285942078, 0.2879190742969513, -0.28287097811698914, -0.6640811562538147, -1.0840415954589844, -1.4510055780410767, -0.6099615693092346, 1.1642390489578247, 0.6080286502838135, -1.2763755321502686, -0.815483570098877, 0.012937751598656178, -1.3997114896774292, 0.476039856672287, -0.7663651704788208, -1.131555199623108, 0.37584567070007324, -0.38665181398391724, -0.7797247171401978, 1.3165054321289062, 0.7150105834007263, -0.12572605907917023, 1.8377037048339844, 0.5172181725502014, -0.8932492136955261, 0.9141892790794373, -1.1249098777770996, 0.34888583421707153, -1.1473352909088135, -0.5339008569717407, 0.08098861575126648, -0.5418114066123962, 1.194048285484314, -0.6827269196510315, 0.5640448927879333, -1.1043683290481567, -1.1109519004821777, -1.4444977045059204, 0.7654449343681335, -1.1205848455429077, 0.03609835356473923, -1.1863361597061157, -0.3723691403865814, 2.5069212913513184, -0.9014427065849304, -0.010733517818152905, -0.7491488456726074, -1.673244833946228, -0.058722421526908875, 1.3084776401519775, 0.5791525840759277, -1.1963688135147095, 0.5123059749603271, -0.018296372145414352, -0.9009177088737488, -0.7071473598480225, 0.3344017267227173, -0.9285868406295776, 0.604314386844635, -0.7556869983673096, -0.7638963460922241, -1.8139585256576538, -0.20729240775108337, 0.1713610291481018, -1.1101211309432983, 0.35499224066734314, 2.1130404472351074, 0.5457817316055298, 0.29807862639427185, -0.2989630401134491, -1.1633163690567017, 1.6385232210159302, 1.0447691679000854, -0.3282286822795868, -0.24362806975841522, 0.7392802238464355, -0.7260721921920776, 0.2652139663696289, -0.36765700578689575, -0.38529473543167114, -0.7274574041366577, -0.10313745588064194, 0.755632221698761, -0.565645694732666, 0.6710500717163086, 1.4561009407043457, 0.04779337719082832, 0.28591564297676086, -0.9362730383872986, -0.9605563879013062, 1.2764160633087158, 0.8178997039794922, -1.70195472240448, 0.4238298833370209, 0.547015368938446, 1.8003536462783813, 1.882920503616333, 0.6023762226104736, 0.04021003469824791, 0.11519969254732132, -0.1858210563659668, -1.3174118995666504, -0.9364780187606812, 0.6905913949012756, 1.0259878635406494, -0.4498218894004822, -0.020290249958634377, -0.19217082858085632, 0.7806271314620972, -1.2964636087417603, -1.1143440008163452, 0.39930564165115356, -0.6802836656570435, 0.5852633714675903, -0.6662877798080444, -2.2243106365203857, -0.5581682920455933, 0.3313143253326416, -0.8807323575019836, 0.3176337778568268, 0.4291019141674042, 0.6256955862045288, -0.5126813650131226, -0.6925081014633179, -0.9402305483818054, 1.356242060661316, -0.5209975838661194, 0.6978026628494263, -0.7404745817184448, 0.5999962091445923, -0.6489931344985962, 0.30410951375961304, 0.41508346796035767, 0.19001798331737518, 1.0298062562942505, -0.5011597275733948, -1.103550672531128, 0.9755099415779114, -0.7315713167190552, -1.0167049169540405, -1.1150776147842407, 1.228690266609192, -0.1633719503879547, 0.7293694019317627, -1.109083890914917, -0.91447913646698, 0.4167501628398895, -0.46667909622192383, 0.2397875189781189, -1.7838850021362305, 0.8981891870498657, 0.8595256209373474, 0.28291162848472595, -1.0104905366897583, -0.012897811830043793, -0.6071439981460571, 0.6671633720397949, -0.3915247321128845, -1.2243075370788574, 1.0737322568893433, -0.7088629007339478, -0.39108291268348694, -0.7293868660926819, 0.035266585648059845, -0.3756044805049896, 0.7791587114334106, 0.5118021965026855, 0.16224871575832367, -1.5914027690887451, 1.0831509828567505, 1.0579992532730103, -0.8856355547904968, 1.651246428489685, 0.5806810855865479, -0.8297207951545715, -1.0925592184066772, -0.6927865147590637, 0.3999442458152771, -0.8998295068740845, -1.1093783378601074, -0.8987548351287842, 0.8302782773971558, 0.36715471744537354, 0.5664175748825073, 0.3632062077522278, 0.42489126324653625, 0.11341769248247147, -1.0511281490325928, 1.136531114578247, 2.0524895191192627, 0.6990246772766113, 0.10582269728183746, 0.44839054346084595, -0.9445815682411194, 0.25349605083465576, 0.5241756439208984, 2.745778799057007, 0.9877811074256897, 0.7162836194038391, -1.357601523399353, 0.11221291124820709, 0.9106202125549316, -0.8257325291633606, 0.44835734367370605, 0.040292464196681976, -1.4027425050735474, -0.7437624931335449, -1.6457525491714478, -0.6885963082313538, -0.20196405053138733, 0.4891211986541748, 0.05728217959403992, -1.0832775831222534, -0.25881797075271606, 0.9480407238006592, 0.3404516875743866, -0.6867353916168213, -0.9800427556037903, -1.2478872537612915, -0.5546261072158813, 0.2782139182090759, 0.2517441511154175, 0.18061643838882446, 0.5131638050079346, 1.046860933303833, 1.3639386892318726, -0.08927197754383087, -0.1752062886953354, 0.20506782829761505, 0.4075930118560791, 1.2428128719329834, 0.3606448173522949, 0.3953884243965149, -0.6717031002044678, 0.5533689260482788, -0.398336797952652, 0.044821131974458694, -0.762813150882721, 0.3597277104854584, 0.5536232590675354, -0.834118664264679, -1.3661608695983887, 1.1336947679519653, -0.43964314460754395, -0.9109078049659729, -1.0954450368881226, -0.8483335375785828, 0.24686388671398163, -1.3798449039459229, 0.47163981199264526, -0.3179894685745239, -0.14943112432956696, -1.0664387941360474, 0.9690598249435425, -0.24091239273548126, -1.3868948221206665, 1.2367513179779053, -0.5125247836112976, -1.0653003454208374, 0.9165935516357422, 0.15250511467456818, 0.7963941693305969, 0.38462382555007935, 0.5051264762878418, 0.11613061279058456, 0.5756656527519226, -0.31058552861213684, 1.1039283275604248, 0.8974552154541016, 0.7989572286605835, -0.705194354057312, 1.5927711725234985, -2.7810332775115967, -0.9503606557846069, -0.6811239719390869, 0.9406903982162476, -0.8859739303588867, -1.1439436674118042, -0.7869173884391785, 1.4744035005569458, -1.2627434730529785, 0.8431912660598755, -0.13038797676563263, 0.776045024394989, 1.3727893829345703, 0.708221435546875, 0.23788028955459595, -0.24592353403568268, 0.6058210134506226, -0.5706672072410583, -1.3908559083938599, 1.5197410583496094, -0.6439883708953857, -1.47742760181427, 0.14032770693302155, -1.9667186737060547, -1.8705817461013794, 1.5266684293746948, 0.08575177937746048, 0.9942030310630798, 1.1259520053863525, 0.475819855928421, 1.0885870456695557, 0.932101309299469, 0.025548258796334267, 0.7501230835914612, 0.8453670144081116, 0.36733901500701904, 0.7521599531173706, -0.5247064828872681, -0.30323174595832825, 0.8193495273590088, -0.3536914885044098, 0.9570022225379944, -1.757744312286377, -0.09246435761451721, -1.3920741081237793, -0.055699173361063004, 0.31381163001060486, 0.3742940127849579, -0.8266443610191345, -0.3756233751773834, -0.8348450064659119, 1.0366768836975098, 0.051495034247636795, -0.8091335296630859, 2.34431529045105, 0.6889848709106445, 1.1585493087768555, 0.18644043803215027, -0.6001063585281372, 0.3426147699356079, 0.37845784425735474, 0.8415972590446472, 0.9173374176025391, 0.8051710724830627, 1.76382315158844, 1.718579888343811, -0.7006882429122925, 0.9580403566360474, 0.551308810710907, 0.9561623334884644, -0.1339206099510193, 0.2368113249540329, -0.2654437720775604, -0.4194931089878082, 1.1227649450302124, -1.5808279514312744, 1.2609144449234009, -1.3400638103485107, 0.6416328549385071, 1.6057229042053223, 1.1092932224273682, -0.011202084831893444, -0.5223045349121094, 0.8844388723373413, -0.5431160926818848, -2.085341215133667, -0.5601313710212708, -0.09552536904811859, -0.8575252890586853, -1.7866215705871582, -2.4499599933624268, 0.19123248755931854, 0.019954293966293335, -0.1744663566350937, 0.34422698616981506, -1.2496528625488281, 0.09850858151912689, 1.7092267274856567, 0.5083096623420715, -0.23653291165828705, -0.3052033483982086, 0.1519748568534851, -0.585347056388855, -1.1586647033691406, -0.6355292797088623, 0.31930482387542725, 0.5536285042762756, 0.16556596755981445, 0.4234961271286011, 1.4418110847473145, -1.1869094371795654, 1.0139127969741821, -0.20752693712711334, -1.2101720571517944, 0.19853900372982025, -1.4126759767532349, -1.5198297500610352, -0.6201512813568115, 0.6096950173377991, 0.20385845005512238, 0.4033462703227997, 0.7036318778991699, -0.03406810015439987, 0.8433997631072998, 0.753788411617279, -0.7363230586051941, 0.33070117235183716, 0.6192669868469238, 0.5765236020088196, -0.693226158618927, 0.641408383846283, -0.4258989691734314, -1.1973206996917725, -0.2139645367860794, -1.6305450201034546, -0.8052499294281006, 0.7870755195617676, -0.5555447340011597, 0.5977160930633545, 0.9883541464805603, -0.011848978698253632, 1.3864248991012573, -0.3119726777076721, -0.3392459750175476, -0.08460233360528946, -1.391748309135437, -0.07130104303359985, -1.3693677186965942, 0.024210520088672638, -0.10629243403673172, 0.2695007920265198, -0.24067194759845734, 0.7811826467514038, -0.33158037066459656, -0.8477121591567993, -2.5738632678985596, -0.28173840045928955, -0.5745775103569031, 1.2950332164764404, -0.0407455749809742, 1.2583438158035278, -1.4478251934051514, 1.2157140970230103, 1.1924442052841187, -1.2453354597091675, 0.8134022951126099, -0.9709539413452148, 1.256536602973938, 0.36783158779144287, -0.5255514979362488, -0.2495722770690918, -0.23340991139411926, -2.7274270057678223, -0.6135960221290588, -1.1317987442016602, -0.039244089275598526, -1.6001781225204468, 1.3765331506729126, -0.8153760433197021, 1.8854315280914307, -1.7756427526474, 0.5172394514083862, -0.7441559433937073, -0.583049476146698, -1.818200707435608, -0.4394465386867523, 0.4293420612812042, 1.7194409370422363, -0.0979803055524826, 0.3625456690788269, -1.3484470844268799, 0.5578228235244751, -0.11671958863735199, -0.15775269269943237, 1.8618204593658447, -0.23998405039310455, 0.044894393533468246, 1.004194736480713, 0.9927381873130798, -0.5725938677787781, 0.2667810916900635, -1.5378303527832031, 0.1058628112077713, 0.6825879216194153, 0.9182791113853455, 0.8046823143959045, 1.7542661428451538, 0.5455368161201477, -1.2879254817962646, 2.5413155555725098, 0.6188087463378906, 0.7152823805809021, 0.12415114045143127, -0.1638566255569458, 0.4577173888683319, 0.6848652958869934, -0.4270596504211426, 0.5501247644424438, -0.03799639642238617, -1.0109397172927856, 0.37713974714279175, -1.462584137916565, -0.7588370442390442, -0.9489519596099854, 0.9727094769477844, -0.10601865500211716, -0.5915068984031677, 0.7110119462013245, -1.4191715717315674, 0.9331048130989075, 0.6153422594070435, 0.4550900459289551, 0.011979229748249054, 0.5881093144416809, -0.9101948738098145, -0.37475061416625977, -0.655463457107544, 0.6951029896736145, 1.5605412721633911, -2.9884769916534424, 0.24025721848011017, 1.730141520500183, -0.652056872844696, -0.37875479459762573, -0.795114278793335, -3.041813611984253, -0.6115940809249878, -0.6344005465507507, -0.34311720728874207, 0.08180844038724899, 1.3372033834457397, -0.43436869978904724, 1.4451450109481812, -0.24000762403011322, -1.2403136491775513, 0.3377074599266052, 1.1011961698532104, -0.4729233384132385, 2.3813071250915527, -1.1856988668441772, 2.6641197204589844, -0.014143994078040123, -1.5206623077392578, 1.1172951459884644, 0.6202581524848938, 1.2149879932403564, -0.34588438272476196, -1.0801010131835938, -0.2734650671482086, -0.5221775770187378, 1.6870083808898926, 0.31752973794937134, -1.797171711921692, -1.4236918687820435, 1.463073492050171, 1.1491113901138306, -1.360837459564209, 1.2277133464813232, 0.014704459346830845, 0.6308974027633667, -1.0898773670196533, -0.6601343750953674, -1.5016844272613525, 1.028037428855896, -1.9807963371276855, 0.5534371137619019, 1.1399492025375366, 0.4571257531642914, -1.0868381261825562, -0.28017523884773254, 0.17733091115951538, -0.02528379298746586, -0.03640081360936165, 0.5610621571540833, -0.1814226359128952, 0.6547968983650208, -1.2926510572433472, 0.6087551712989807, 0.8697108626365662, -0.6136215925216675, 1.6909382343292236, -0.10203544050455093, -0.03281230479478836, -0.3027600646018982, -0.902862012386322, 0.5779181122779846, -1.6328046321868896, 1.0560487508773804, 0.39704227447509766, 0.4240185618400574, 1.279576063156128, -0.18815447390079498, 0.24882762134075165, 0.05259474366903305, 1.0747771263122559, 0.6576488614082336, 0.9309167861938477, 0.5499337315559387, 0.6847628951072693, -0.12991996109485626, -0.8010626435279846, 0.04281160980463028, 0.5115683674812317, 0.7439501285552979, 1.5721795558929443, 1.5865064859390259, -0.4746279716491699, -2.9995052814483643, -0.07085701078176498, -0.9431581497192383, -0.1280728429555893, -0.7729506492614746, 2.487941026687622, -0.3718389868736267, 0.3754098117351532, -1.000441551208496, -0.1595376431941986, 0.28833863139152527, -0.41724708676338196, 0.711845338344574, -0.7474841475486755, -0.16682438552379608, 0.024531258270144463, -1.2082892656326294, 1.7805399894714355, -0.1547742336988449, 0.04541540890932083, -0.9873207807540894, 0.5510994791984558, -0.006023051217198372, -0.6365906596183777, 0.2673409581184387, 0.506907045841217, -0.5205901265144348, 0.4333248734474182, 1.6054539680480957, -0.8289263248443604, 1.0276693105697632, -0.7510762810707092, 0.740225613117218, 0.3900023400783539, -1.1040304899215698, 0.02984737418591976, 0.3223731815814972, 0.3487786650657654, 0.13589297235012054, -1.4149173498153687, -0.0923331007361412, -0.6685636043548584, 0.25079184770584106, 1.4200313091278076, -0.21106499433517456, 0.845643162727356, -0.4367358386516571, 0.33340656757354736, -0.1758541464805603, -0.030483154579997063, 0.009251553565263748, 0.4766632914543152, -0.2118925005197525, -0.23702339828014374, -0.30469179153442383, 0.6634044647216797, -0.49590927362442017, 0.487531840801239, -2.618622064590454, 0.5503768920898438, 0.18399125337600708, 1.0045220851898193, -1.2172143459320068, -1.8132587671279907, -0.6133819818496704, -0.5997657775878906, 1.0126243829727173, 0.9287915229797363, 0.23147809505462646, 0.5598711371421814, -0.668855607509613, 0.06634841114282608, 0.5454308390617371, -1.6053600311279297, 1.2689597606658936, -0.026859741657972336, -0.24641700088977814, -0.5765307545661926, -0.7658358812332153, 1.3285608291625977, 0.5252127051353455, 1.3601529598236084, 1.0723073482513428, 2.4492528438568115, -1.1505067348480225, -0.31150394678115845, 2.13273549079895, -1.2100636959075928, -0.8013660907745361, -0.8203498721122742, 1.2995100021362305, -0.2557196319103241, -0.6173444986343384, -0.6909571290016174, 0.8946624398231506, -0.15893873572349548, -1.0112398862838745, 0.15201526880264282, 1.2170045375823975, 2.0008254051208496, 0.16550737619400024, -0.12198911607265472, -0.6414411067962646, -0.9664655923843384, 0.7089617252349854, -1.5646775960922241, 1.2284899950027466, 1.0421499013900757, 1.0052976608276367, 0.4727323353290558, -0.5448367595672607, 0.8839803338050842, -0.818818986415863, -0.051470063626766205, 0.1242288127541542, -1.2545716762542725, -1.52375066280365, -0.8164129853248596, -0.640791654586792, 1.1738841533660889, -0.7137653231620789, 0.19625285267829895, 0.3759799599647522, -0.7837362289428711, -0.2322702556848526, -1.194670557975769, 0.050367020070552826, -1.413618803024292, 0.2517487406730652, -1.0328528881072998, 0.0075348359532654285, -0.6209498047828674, -1.1252796649932861, -1.929701328277588, -0.31913092732429504, -0.36350345611572266, 0.5966349840164185, -0.42879718542099, -1.0834909677505493, -1.3284471035003662, 0.01989736221730709, 0.5866215825080872, -0.233059823513031, 0.15945129096508026, 0.10883761197328568, -0.932816743850708, -0.5493147969245911, 0.6769318580627441, 1.1653214693069458, 1.5451247692108154, 0.889481782913208, -0.2831852436065674, 0.3709172010421753, 0.2177562564611435, 0.9785087704658508, 1.3016772270202637, -0.1760862022638321, -0.45203909277915955, -1.9157289266586304, -0.34089258313179016, -0.6327136158943176, -1.0839619636535645, 1.7304801940917969, 0.24234311282634735, -0.4739902913570404, -0.02628384716808796, 1.0958200693130493, 0.046244122087955475, 3.0730104446411133, -1.6722966432571411, -0.7067393660545349, -0.15120165050029755, -1.849442481994629, -1.053385615348816, -0.7664182782173157, -1.8200857639312744, 0.976495087146759, -0.1349431425333023, 0.9463687539100647, -0.3145109713077545, -1.626172423362732, 0.20846788585186005, -1.3819079399108887, 0.38996952772140503, -0.1013910323381424, 2.5336155891418457, -0.4568047523498535, 0.2755826711654663, -0.5743971467018127, 1.052528977394104, 0.9496831297874451, 0.3587920367717743, -1.8601760864257812, 1.2595231533050537, -0.9269847869873047, -0.832190752029419, -0.3874446749687195, -0.6008397340774536, -0.6429058909416199, 0.4201688766479492, -0.10143402218818665, 0.8958252668380737, 0.580780565738678, 1.2264299392700195, 0.5228645205497742, 0.17036142945289612, 0.1084221601486206, -0.5522473454475403, 0.8669673204421997, -1.6946394443511963, 0.7338337302207947, 0.25258827209472656, -1.490369200706482, -0.9255403280258179, -1.6094272136688232, -0.0040288567543029785, 2.1040050983428955, -0.09132835268974304, 0.8276404142379761, -0.8816408514976501, 1.3827415704727173, 1.2283880710601807, 2.3561553955078125, 1.2621647119522095, 1.3283501863479614, -0.7579533457756042, -0.6089736223220825, 0.4588480591773987, 0.02185158059000969, -1.1243305206298828, -0.6510381102561951, 0.8511555790901184, -0.6725187301635742, 1.56136155128479, -1.4088108539581299, -1.1772878170013428, 0.12520617246627808, 0.4084317088127136, 1.7182832956314087, -1.212969422340393, -1.6592501401901245, -0.8077018857002258, 0.2510957717895508, 0.8031343817710876, 0.5791970491409302, 0.33169037103652954, 1.7213070392608643, -0.6222869753837585, 0.4687751233577728, -0.26377832889556885, -0.2682102620601654, -1.3828346729278564, -0.24301929771900177, -0.7261354327201843, -0.3224778473377228, 0.22809474170207977, -1.3533071279525757, 0.36921238899230957, 1.1585246324539185, 0.46535032987594604, -0.24671652913093567, 1.3551565408706665, -0.8288372159004211, -0.07590228319168091, 1.5242983102798462, 0.9905869364738464, 0.3144702911376953, 0.12309173494577408, -0.29553309082984924, -0.19631430506706238, -0.40870338678359985, -0.7732446193695068, 1.4606339931488037, -0.37514713406562805, -0.3495158553123474, 0.9702529311180115, 1.2397197484970093, 1.6169863939285278, -0.23544353246688843, -0.1019776239991188, 0.5564287304878235, 0.6160146594047546, 0.09373930096626282, 0.1684839427471161, -2.178696632385254, -0.46204110980033875, 0.16554032266139984, 1.3004614114761353, 1.362127423286438, -0.6684619188308716, -1.0207791328430176, 0.42988091707229614, 1.3495639562606812, 0.003738188650459051, -1.5698801279067993, -0.4758774936199188, 0.17752614617347717, 2.3615574836730957, 1.1563735008239746, 1.4174485206604004, 0.9806144833564758, -0.7741418480873108, 0.08796198666095734, 0.6424298882484436, 0.9499463438987732, 0.7737336158752441, 1.108988642692566, -0.5573720932006836, -0.9225369095802307, -0.09081776440143585, -0.4706919193267822, 1.6083695888519287, 0.2109655886888504, -0.1878734976053238, -1.2089897394180298, -1.6951957941055298, 2.352855920791626, 0.43728432059288025, -0.7947864532470703, -0.45442456007003784, -2.9123332500457764, 0.8233930468559265, 0.9630124568939209, -1.0338844060897827, 0.9508525729179382, 0.7367770075798035, 0.5057782530784607, -2.1377406120300293, 0.07345308363437653, 0.6674544811248779, -2.6279079914093018, 0.5957284569740295, -0.342776358127594, -0.06744101643562317, 0.551462709903717, 0.8905335664749146, 0.733429491519928, -0.4373943507671356, 0.3698451817035675, 0.1192471981048584, 1.7786141633987427, 2.0624279975891113, -0.08794530481100082, 1.5712345838546753, -1.1185110807418823, -0.4088194966316223, 2.9072377681732178, 1.2160207033157349, 0.5655978918075562, 1.0911972522735596, -0.8460800051689148, -0.24354198575019836, 0.5510686635971069, 0.45007672905921936, 0.6981039047241211, 0.6476631164550781, -0.2557515799999237, -0.32739850878715515, -0.5072495341300964, -2.7166783809661865, 0.6067299842834473, -1.2654894590377808, -1.2561477422714233, -0.04338627681136131, 0.19327671825885773, -0.1485832780599594, 0.9215255975723267, -1.7189245223999023, 0.8562259078025818, 0.19047129154205322, -0.5418276190757751, 0.6291272044181824, -0.15797118842601776, 0.2933815121650696, -0.4406997263431549, -2.082035541534424, -2.4562103748321533, 0.6088326573371887, 0.07797765731811523, -1.7018407583236694, 0.3440151810646057, 1.928057312965393, -3.646383285522461, 0.6275460124015808, 0.25353899598121643, -0.2162286341190338, -0.3346472382545471, -0.6705577969551086, -0.38960954546928406, 0.3953460454940796, -0.26114755868911743, -1.6169506311416626, 0.42849284410476685, -0.2722712457180023, -0.7264434099197388, -0.2415543794631958, 0.5214167237281799, -0.6940065026283264, -0.7057769298553467, -2.034045696258545, 0.14800521731376648, 0.16609331965446472, 0.017320264130830765, 0.45290786027908325, 3.0859873294830322, -1.8133233785629272, -0.1996205747127533, -0.6217076778411865, -0.9151198267936707, 1.4870705604553223, 0.5199082493782043, -0.8525422811508179, 0.15916018187999725, -0.5409398078918457, 2.335566759109497, 0.03942369669675827, 1.700084924697876, 0.7673683762550354, 1.0664726495742798, -0.5454893112182617, 2.2521822452545166, -2.1740031242370605, -0.9950198531150818, -0.7766912579536438, -0.031301312148571014, -0.47806820273399353, 0.29932618141174316, 0.5334200263023376, -1.8320858478546143, 0.994857132434845, 0.2720867991447449, 0.24208489060401917, 1.3561391830444336, -0.6274566054344177, 0.18580792844295502, 1.1337674856185913, -0.6812971830368042, 0.46606603264808655, 1.3637775182724, -0.03275166451931, -0.5023665428161621, 1.0416457653045654, -0.6397464871406555, -1.1517398357391357, 0.6852307915687561, -1.2309664487838745, 0.102530837059021, 1.340901255607605, -0.08900976181030273, -0.028034983202815056, 0.5534605383872986, -0.18840216100215912, 0.9610251188278198, -0.7854443192481995, -0.721024751663208, -0.5554687976837158, 0.5044736266136169, 1.943782925605774, 2.6400814056396484, -0.8220414519309998, 0.972749650478363, -0.31512752175331116, 0.23188695311546326, -0.5922015309333801, -0.6191376447677612, -0.45566320419311523, -1.2932018041610718, 0.9008610844612122, 0.2710401713848114, 0.12097728252410889, -0.7589552402496338, 0.6146038770675659, 0.4847136437892914, -0.36771148443222046, -0.2339710295200348, -0.06895336508750916, 0.7460420727729797, -0.11287185549736023, 1.8606880903244019, -0.23742513358592987, 0.6540504693984985, -0.872912585735321, 0.43658220767974854, 0.7498937249183655, 1.5218960046768188, 0.1791726052761078, 0.03376566618680954, -1.3149060010910034, -1.1335078477859497, 0.4301122725009918, -0.45576849579811096, 0.7354533076286316, 2.147580862045288, 0.9624276161193848, -0.5241925716400146, -0.25622251629829407, 0.10999522358179092, -0.7350857853889465, -0.13791972398757935, -1.3598607778549194, 1.2729216814041138, -1.0153627395629883, 1.0827454328536987, -0.09943845868110657, -0.049479126930236816, 1.2194041013717651, -0.47756725549697876, -0.7345470190048218, -2.0118448734283447, 0.0721888616681099, -0.7081434726715088, -0.37993213534355164, -0.5983580946922302, 1.4213104248046875, -1.7390345335006714, -0.3815070688724518, 0.15364040434360504, -2.091120958328247, 1.3678656816482544, 0.8766700625419617, -0.38740065693855286, 1.0841395854949951, 1.299129843711853, -1.8485448360443115, 0.22718515992164612, -0.5280473828315735, 1.6053279638290405, -0.29590684175491333, -1.0033358335494995, 0.5214439630508423, 0.038541991263628006, -0.5581493377685547, -1.6973986625671387, -0.23720456659793854, 0.3034067451953888, -1.5255461931228638, 0.8585100769996643, -0.18317773938179016, 0.655582070350647, -0.999895453453064, -1.0119112730026245, 0.005626744125038385, -1.535007119178772, -0.3619943857192993, -1.1933701038360596, -0.8289826512336731, 0.35208532214164734, 1.7449568510055542, 0.3627384603023529, -0.5388125777244568, 0.9085321426391602, 0.6153612732887268, -1.5689220428466797, 0.3303981125354767, 0.535760760307312, 0.37164250016212463, -0.9214398264884949, -0.1998578906059265, 0.3498803973197937, 0.875204861164093, 0.5466601848602295, 0.02121555805206299, 0.4919358789920807, -0.6402491927146912, 0.7377257943153381, 0.07581520825624466, -0.7913213968276978, 0.68468177318573, -0.8763818144798279, 0.33719339966773987, -0.1804867386817932, -0.8741969466209412, -1.0704792737960815, 0.6536397337913513, -0.917790412902832, 0.5974849462509155, 1.0954933166503906, -0.7098820805549622, 0.2611044943332672, 0.4552181661128998, 0.5752840042114258, 2.6424756050109863, -0.8104353547096252, -1.6620724201202393, -0.07633587718009949, 0.5330529808998108, 1.0045284032821655, -0.01992897316813469, -0.1114872395992279, -0.5132520794868469, 0.21396712958812714, -0.01695886440575123, 1.3677868843078613, 0.8064380884170532, 0.21633876860141754, 0.5078577995300293, 1.554946780204773, -0.6996679306030273, -0.2451988309621811, 0.14959822595119476, -0.30801841616630554, -0.2068624198436737, 1.8451203107833862, -0.9291003942489624, -0.6820762157440186, 1.0334354639053345, -0.545914351940155, 0.9174328446388245, -1.0962955951690674, -0.5492466688156128, 0.31963926553726196, 0.5908272862434387, -0.7812289595603943, -0.0006924363551661372, -0.6053171753883362, -1.8760420083999634, -1.2760212421417236, -2.3682520389556885, -1.2443645000457764, -0.12091527134180069, -0.42253562808036804, -0.5326854586601257, -0.18786272406578064, 1.3614065647125244, -1.0506455898284912, 0.29545271396636963, -0.008370568044483662, 1.0009692907333374, 1.445560097694397, 0.3365512490272522, 0.7164299488067627, 0.8510740399360657, 0.30667775869369507, -0.5367104411125183, -0.29687339067459106, 0.37220498919487, 1.107770562171936, -0.08549242466688156, -0.09926599264144897, -1.3419429063796997, -0.32569125294685364, -1.5387555360794067, 1.980282187461853, 1.2818433046340942, -0.5204681158065796, 0.09467703104019165, 1.0660243034362793, 2.185028553009033, -0.8047422766685486, 0.631544291973114, 0.12168838083744049, 0.8141267895698547, -0.38839206099510193, 1.8978769779205322, -0.6163029670715332, -0.7996615767478943, 0.06469537317752838, -0.12877558171749115, -0.5457587838172913, -0.504565954208374, 0.19016584753990173, -1.462697148323059, -0.38683080673217773, -1.0670361518859863, -0.6475410461425781, -0.3693650960922241, 0.49729758501052856, 0.25071921944618225, -0.012635289691388607, -1.3278292417526245, -0.3618839979171753, -1.7264519929885864, 0.7722419500350952, -0.04737715795636177, 0.5733880400657654, -2.2594661712646484, -0.3971514403820038, -0.6560940146446228, 1.188966155052185, -0.9667336940765381, -0.14099013805389404, 1.6467339992523193, 0.8947060108184814, 0.15121643245220184, 0.136562317609787, -0.9111592769622803, -0.49892643094062805, 0.37475305795669556, 0.890163779258728, -1.507608413696289, -0.1176973357796669, -0.024224525317549706, 0.4888222813606262, -1.8648205995559692, 0.9492988586425781, 0.8289737105369568, 0.7621864080429077, -1.9336957931518555, 1.1110703945159912, 0.8931744694709778, 0.27846693992614746, -0.9224399924278259, 0.6757916808128357, -0.06948711723089218, 0.9146401286125183, -0.6327826380729675, 1.2213777303695679, 1.19767427444458, 1.6983460187911987, 0.39369699358940125, 1.5912262201309204, 0.8754065036773682, 0.468038409948349, -0.5613455176353455, -1.6854350566864014, -0.1545512080192566, -1.4440187215805054, -1.057550311088562, -1.6881728172302246, -1.3439159393310547, 1.3364629745483398, -0.5862758159637451, 0.8111699223518372, -0.13703566789627075, -1.0096571445465088, -2.1504995822906494, -1.0901093482971191, 0.5835879445075989, -1.0373064279556274, 2.7333686351776123, -1.2819972038269043, 0.38678672909736633, -0.04785386100411415, 0.297140508890152, -0.175027996301651, 0.6308535933494568, -1.1391831636428833, -0.7673720717430115, 0.795102059841156, -0.11278115212917328, -1.9812965393066406, -0.8194831609725952, -0.5696008801460266, 2.966017961502075, 0.2934616208076477, -1.889559268951416, 1.3208204507827759, -2.0208945274353027, -0.7679685950279236, -0.8735349178314209, 0.15456253290176392, 0.9277178645133972, -0.9870839715003967, 2.225921154022217, 0.5029664039611816, -1.606355905532837, -0.6604821681976318, 2.4682106971740723, -1.2420564889907837, 0.4402504861354828, -0.27044612169265747, -0.6012058854103088, 0.27296119928359985, 0.07588103413581848, -1.5715194940567017, 0.583584725856781, -0.8504904508590698, -0.8626601696014404, 0.2780023515224457, 0.6461221575737, 0.7367362976074219, -2.095966339111328, 0.9852147698402405, 0.4214535653591156, -1.4079657793045044, -0.3960179388523102, 1.881564974784851, -0.6252825856208801, 0.690220057964325, 0.142647847533226, 0.7072579264640808, -0.1250341683626175, -1.8813191652297974, 0.23326490819454193, 0.9431881308555603, 0.23909534513950348, -1.3549162149429321, -0.11481447517871857, -0.5531352758407593, -0.7181406021118164, 0.6002616882324219, 1.7477271556854248, -1.6988331079483032, -0.024717628955841064, 2.1737732887268066, -0.6563757658004761, -0.1127157136797905, 2.04317307472229, 1.4414434432983398, 0.6621227264404297, 0.03174750506877899, -0.6933631300926208, 1.077059268951416, -0.04406441003084183, -0.17986734211444855, 0.7151148915290833, -0.004312017932534218, -0.38260847330093384, -1.19960355758667, -1.9098153114318848, 0.6209484934806824, -0.10238657891750336, 1.0027440786361694, -2.0078625679016113, 0.08695585280656815, 0.17407728731632233, 0.944069504737854, 0.8139680027961731, -0.042905043810606, 1.74626624584198, 1.3532633781433105, 1.271863341331482, 1.1747606992721558, 0.033018603920936584, -2.0338082313537598, 0.36723875999450684, -1.567568302154541, 1.469409465789795, -1.6427232027053833, 1.0074496269226074, -0.04896434396505356, 0.13463501632213593, -0.9944908618927002, 2.195594549179077, -0.6946337819099426, 0.2948457598686218, -0.22832798957824707, -0.9857662320137024, 0.4795789122581482, 0.2705838084220886, -0.19248194992542267, 1.7239320278167725, 0.18552863597869873, -1.1192318201065063, -0.5579977035522461, 0.5279721617698669, -1.3188751935958862, -0.3876895010471344, -1.0626353025436401, 0.29005497694015503, 1.3685312271118164, -0.44628989696502686, -1.8271963596343994, 0.21422050893306732, -1.1306533813476562, 0.9149401187896729, 0.3475361168384552, -1.1848247051239014, -0.2805067300796509, 0.6874285936355591, -0.45842501521110535, 0.041113924235105515, 0.8227875232696533, -0.07849199324846268, -0.6891010999679565, -0.7089765667915344, 1.2674510478973389, -0.018617896363139153, 0.35890376567840576, -0.7587856650352478, 2.0432398319244385, -1.374240517616272, 0.19255416095256805, 1.280564785003662, 0.34676334261894226, 0.7458973526954651, -0.8371214866638184, 1.9486708641052246, 1.3149887323379517, 0.8621126413345337, 3.205854892730713, -0.23869122564792633, 0.11576338112354279, 0.8602980375289917, -1.3978127241134644, 1.0811630487442017, -0.40842247009277344, 1.9114147424697876, -0.9320237636566162, -2.2278943061828613, 0.9877902269363403, 0.1845969706773758, 1.3592438697814941, 0.677650511264801, -0.5498258471488953, 2.8045156002044678, 0.6374300718307495, 0.4461861550807953, 0.27032041549682617, 0.08697416633367538, 1.985813021659851, 1.9556043148040771, -0.7626497149467468, -0.07054102420806885, -0.25366055965423584, -0.7895184755325317, -0.2400222271680832, 0.16580168902873993, 0.9472194314002991, -0.7321400046348572, -0.7732776999473572, -1.9881433248519897, -0.2690691351890564, -1.6750017404556274, -0.7473854422569275, -1.4148871898651123, 0.4633086323738098, 0.4234810769557953, 0.15704604983329773, 0.5006348490715027, 0.917682945728302, 0.8809357285499573, -1.3021348714828491, 1.6123024225234985, 0.6577328443527222, -1.8775498867034912, -0.8215345144271851, 0.33565330505371094, -0.8805349469184875, 0.5684415698051453, 0.12199317663908005, 0.8866162896156311, 1.4007924795150757, -0.1670561134815216, 0.6191201210021973, 0.3061274290084839, -0.02780984528362751, -0.1331252157688141, -0.5341290831565857, -0.6607625484466553, -1.7179425954818726, 2.243581771850586, -0.059488940984010696, 2.153646230697632, -0.45717939734458923, 1.3474655151367188, 1.4245525598526, 0.3986486494541168, 1.6877315044403076, 0.896712064743042, 1.075204610824585, -2.1746978759765625, -0.2660056948661804, -0.7392326593399048, 1.7730456590652466, -1.8727363348007202, -1.6787983179092407, -1.1348521709442139, 0.4688279628753662, 1.675209879875183, -0.3988015651702881, 0.8316473364830017, -0.009796682745218277, 0.5946887135505676, 0.22352808713912964, -0.4755377173423767, -0.15993303060531616, -0.1665695309638977, -1.899480938911438, 0.4701750576496124, 1.0778130292892456, 1.8389321565628052, 0.09782421588897705, 0.010248116217553616, -0.5356059074401855, 1.1686530113220215, -1.001050591468811, 0.48976069688796997, -0.10770723223686218, 0.6378560066223145, 0.4838123917579651, 0.6206671595573425, 0.4970192015171051, -0.6986645460128784, 1.2864660024642944, -0.581662118434906, -0.4661073088645935, 0.4052431285381317, -1.2723257541656494, 1.3533670902252197, -1.9641629457473755, -0.28253111243247986, -1.008435845375061, -0.6608715653419495, 1.1275200843811035, -0.6632491946220398, 0.6175239682197571, 0.3747386932373047, -0.08734264224767685, -1.6958537101745605, 0.5998396277427673, 0.3464809060096741, -0.8178759813308716, 0.6103785037994385, 0.3822830319404602, -0.2834344804286957, 0.031297653913497925, -0.1811293065547943, -0.9688237905502319, 1.0564264059066772, -1.3656907081604004, -1.1651686429977417, -0.290399432182312, -0.7336518168449402, 0.44531136751174927, 0.3837604820728302, 0.2182234674692154, -0.5539525747299194, -1.308644413948059, 1.0258967876434326, -0.3477945625782013, 0.31907275319099426, -1.020552635192871, 1.4928094148635864, 0.5828962922096252, 0.3826081156730652, -4.99130192110897e-06, 0.8356978893280029, -0.941302478313446, 0.1316681206226349, 0.004800340626388788, -0.35426533222198486, -1.555602788925171, 1.2449638843536377, 0.11977839469909668, 1.5520228147506714, 1.140248417854309, 1.8428456783294678, 0.7303664684295654, 0.4109291434288025, 2.3175132274627686, 2.842346429824829, 0.11094266921281815, 0.6022139191627502, 1.072853446006775, 1.593121886253357, 0.4955708682537079, 0.47858762741088867, 0.12444622069597244, 2.619117498397827, 0.1098056435585022, 0.23182158172130585, -0.6206421256065369, 1.393967628479004, 0.06246666610240936, -0.9400607943534851, 0.17284055054187775, -1.3818875551223755, -0.1487981528043747, -0.1522059440612793, -0.5319564938545227, 0.034649141132831573, 0.4343132972717285, -0.9500914812088013, 1.7429649829864502, 2.791496515274048, -0.5243244767189026, -1.485211968421936, -0.8155461549758911, 0.07508682459592819, 0.47800213098526, 0.7246175408363342, -0.16830213367938995, 1.0292882919311523, -1.0500061511993408, 0.5490468144416809, 1.3563015460968018, -0.9551497101783752, -0.3668025732040405, 0.22518938779830933, -0.9190317988395691, 1.0996431112289429, 0.47402480244636536, -1.2468053102493286, 1.2007197141647339, 1.80012047290802, -0.4444686472415924, -0.5293759107589722, 0.7810843586921692, -0.19277751445770264, -0.5892459154129028, 0.46366265416145325, -0.08756804466247559, -0.3489737808704376, -0.32643812894821167, -0.2061205357313156, -1.1921679973602295, -0.4057457745075226, -0.4911128282546997, 0.389150470495224, -0.6529092192649841, -1.032372236251831, 0.829239010810852, 0.1212267354130745, 0.2751321792602539, -0.4386899769306183, -0.39538881182670593, -0.3668104410171509, -1.4173407554626465, 0.119948610663414, -0.09975643455982208, 1.2825367450714111, 1.75347900390625, 0.7445341348648071, 0.014097141101956367, 0.6469372510910034, 2.2611610889434814, -0.7312763929367065, 0.5843393206596375, 0.5107499957084656, -1.652207612991333, -1.2445021867752075, -2.0324344635009766, -0.006689090747386217, 2.1897342205047607, 0.08121543377637863, 0.653417706489563, 2.1846232414245605, 0.0005263875937089324, 1.4509013891220093, -0.5098444819450378, -1.5207204818725586, 0.05609404668211937, -1.6263914108276367, 0.16893094778060913, 0.30698785185813904, -1.6271991729736328, 0.6425821185112, -1.1627047061920166, -1.0349676609039307, 1.0880556106567383, -1.779098391532898, -1.004172921180725, -0.7656380534172058, 0.5415242910385132, 0.8937953114509583, 0.18631967902183533, -0.568386435508728, -1.5860871076583862, -0.050408415496349335, 0.028719624504446983, 0.6764249205589294, -0.20453791320323944, 0.11683149635791779, 0.9115602374076843, -1.9290800094604492, 0.9498528242111206, -0.7838900089263916, 0.36215537786483765, 1.1455585956573486, -0.6181869506835938, -0.11454424262046814, -1.2642854452133179, -0.8330309391021729, -1.0083597898483276, -0.5362944602966309, -0.6115505695343018, -1.4301958084106445, 1.3390463590621948, -0.40237605571746826, 0.2012699693441391, -1.1040419340133667, 0.9354395866394043, -0.17156633734703064, 0.03940017148852348, -0.03538290411233902, 0.889371395111084, -0.6833318471908569, 0.6000686883926392, 0.0515124686062336, -1.2544023990631104, -0.63474440574646, -1.1018362045288086, 0.019847452640533447, 0.47073614597320557, 2.5463404655456543, 0.081680066883564, -0.28228282928466797, 0.5626208782196045, 1.2166571617126465, -0.16473080217838287, 0.36382269859313965, 0.6236314177513123, -0.34738099575042725, -1.0490657091140747, 0.42075684666633606, -0.8980246186256409, 0.24768131971359253, 0.46711766719818115, -0.9127563834190369, -0.31909701228141785, -0.7570188045501709, -0.5996631979942322, 1.9775350093841553, 1.63361394405365, 0.5610228180885315, 0.24342471361160278, 0.7031792402267456, 2.1933891773223877, 2.68485689163208, 0.7333000898361206, -1.329685926437378, -0.13042838871479034, 0.5449627637863159, -1.1701765060424805, -0.01622883975505829, -0.5435630679130554, 0.5853349566459656, 2.1797375679016113, 0.023100238293409348, 1.0876049995422363, 0.37714576721191406, -2.017129898071289, 0.12461631000041962, -0.5937477350234985, 0.9003349542617798, -0.21991434693336487, -0.41760092973709106, -0.11509008705615997, -0.4770306646823883, -1.060433268547058, 1.9877517223358154, -2.3556835651397705, -0.23209621012210846, -0.24029551446437836, 0.8530930876731873, 0.5159286260604858, -1.2722904682159424, -0.6433835625648499, -1.0732959508895874, -1.1784535646438599, 1.4343583583831787, 1.4880971908569336, 0.5317612290382385, -0.691781222820282, 0.8603777885437012, -1.599987506866455, -1.2025816440582275, -0.013502984307706356, 0.5065587759017944, 0.08797836303710938, -1.8603763580322266, 0.4708300232887268, 1.1455861330032349, 1.1715660095214844, 0.7042254209518433, -1.678159475326538, 1.6500036716461182, 1.7521663904190063, -0.2784985601902008, 1.8735541105270386, 0.8793186545372009, -0.22580713033676147, 0.4133882522583008, -0.36009886860847473, -0.06925033777952194, 0.6265191435813904, -0.6876837015151978, 0.07232082635164261, -0.235920250415802, -0.18947985768318176, -0.46708038449287415, -0.0948929637670517, 0.9320245981216431, -0.5204246640205383, -0.26133596897125244, -0.08813979476690292, 1.2448313236236572, 0.25554636120796204, -0.08819365501403809, 0.8016899824142456, -0.8644577860832214, -0.30904608964920044, 0.8790199756622314, -0.030011247843503952, 0.5766654014587402, 0.5945287346839905, -0.30458733439445496, -0.15618449449539185, -0.11083311587572098, -0.5408675074577332, -1.5829206705093384, -2.932227611541748, 0.6921347379684448, -1.0128984451293945, -0.41508451104164124, -0.4324607253074646, 0.376977801322937, -1.1559617519378662, 0.1511644423007965, 0.5878134965896606, -0.0014163196319714189, 0.020599128678441048, 0.5079149007797241, 1.2438569068908691, -1.2161120176315308, 2.1174633502960205, 0.2106974869966507, 0.1590847223997116, 1.828800082206726, -1.07906174659729, -1.193406105041504, -1.0127333402633667, -1.154320478439331, -1.3055857419967651, -1.1321957111358643, 0.9753618240356445, -1.1372015476226807, 0.30977293848991394, -0.18854789435863495, -0.7403535842895508, -0.9016643762588501, 0.6205554008483887, -0.11584596335887909, -0.9612677097320557, -0.02740432694554329, 2.7979228496551514, -1.3061109781265259, 0.01740924082696438, 0.9928523302078247, 0.6748734712600708, 1.0140758752822876, -0.37330976128578186, -1.1052626371383667, 2.588113784790039, -2.145411968231201, -0.4678625166416168, -1.3520013093948364, -2.042689085006714, -0.07487618923187256, 1.0085405111312866, -0.240152969956398, 0.3198818266391754, -2.3098855018615723, -1.1652265787124634, -0.1030862033367157, 0.3594318926334381, 0.1797354519367218]], "dtype": "float32"}, {"__nd__": [1.4802937507629395, -0.7054319977760315, 0.035795658826828, -1.1728923320770264, -1.2475531101226807, -0.9346947073936462, 0.6959055662155151, -0.2959751486778259, 0.02816811017692089, 0.8880131840705872, -0.6545118093490601, -1.0802212953567505, 0.9634748101234436, 2.009633779525757, 1.3201673030853271, 0.5025981664657593, 1.3155860900878906, 1.005424976348877, 1.4782836437225342, -0.11017537117004395, 2.183588981628418, 0.17818421125411987, 1.1833690404891968, 1.765934944152832, 1.5800814628601074, 1.588168978691101, 1.6019059419631958, 0.3160308301448822, 1.2130979299545288, -0.3929506540298462, -0.34130504727363586, -0.859263002872467, -0.46501484513282776, 0.020520348101854324, -0.9292500615119934, 0.7897677421569824, -0.34645575284957886, -0.7842081785202026, 0.029405346140265465, 0.8405341506004333, -0.9319764971733093, -0.27541497349739075, 2.406822443008423, -0.07408387959003448, 0.9243771433830261, -1.2721627950668335, -0.45781585574150085, -0.9108631610870361, 0.4537131190299988, -0.5887289643287659, 1.570002555847168, -1.8293179273605347, -1.2984037399291992, -1.7459639310836792, 0.12870462238788605, 1.0466187000274658, -0.9931159019470215, -0.770219087600708, 1.573776125907898, 0.5554185509681702, -0.8734275102615356, -1.3713537454605103, 2.314969539642334, -0.7609672546386719, -0.07316701114177704, 0.14547334611415863, 0.5142480134963989, -1.261600375175476, -0.6425105333328247, 0.49648964405059814, -1.126327633857727, 1.9888979196548462, 1.1643139123916626, -1.2111107110977173, 0.9138548970222473, 0.5784264802932739, 0.5970129370689392, 0.200219064950943, 0.13599519431591034, 0.022766491398215294, -1.467330813407898, -0.8628329038619995, -1.5043244361877441, -1.932713508605957, 1.401075839996338, 0.08956154435873032, 0.09583249688148499, -1.2203304767608643, 2.177678108215332, -0.1764339655637741, 0.10204339027404785, 1.5558655261993408, -0.12012360244989395, -1.2229292392730713, -0.3051247000694275, 0.9135539531707764, -0.40783292055130005, 0.21120794117450714, 0.8244239687919617, -0.04641665890812874, 0.9661908745765686, -2.76792573928833, 0.6632980704307556, -1.3460133075714111, 0.18644295632839203, 0.07400917261838913, -1.0356781482696533, -1.001569390296936, -0.0096300533041358, -0.4465441405773163, 0.22137965261936188, 0.6427019238471985, -1.9362517595291138, 0.21867693960666656, -0.41672608256340027, 1.3261339664459229, -0.14415417611598969, -0.43421033024787903, 0.5055971741676331, -0.6517626047134399, -1.7960433959960938, -0.13394159078598022, -0.03675342723727226, 0.3284190595149994, -1.306342363357544, 0.19455654919147491, 0.6289801001548767, -0.3166619539260864, -0.22072826325893402, 0.378892719745636, 0.4196450710296631, 0.047134771943092346, 0.7791458964347839, 0.5592765212059021, 0.27495884895324707, -0.47097986936569214, -0.02278791181743145, -1.007918119430542, 0.04819014295935631, -0.2825874090194702, -0.6388084292411804, 1.2327584028244019, -0.5484961271286011, -2.05289888381958, 1.6786421537399292, 0.36599475145339966, -0.9905301332473755, 1.0942426919937134, -0.12526856362819672, -1.2074470520019531, -1.926566243171692, 0.5193137526512146, 0.38400429487228394, -0.8246327042579651, -1.8159618377685547, 1.502182960510254, -0.37448617815971375, -1.2151362895965576, 1.2052922248840332, 1.562614917755127, -1.4602632522583008, -0.9250308871269226, -0.12466471642255783, 1.6663775444030762, 0.2687014639377594, 1.080823302268982, 0.2707424759864807, -0.08341967314481735, -1.014042854309082, -0.4856501519680023, -1.5927590131759644, -0.10668139904737473, 1.6507198810577393, 0.4351542592048645, -2.657538652420044, -0.20800606906414032, -0.28316712379455566, 0.5895197987556458, 0.9711948037147522, 0.9783328175544739, 0.26449620723724365, -1.600669503211975, -1.41777503490448, -0.8601064682006836, 0.7859649658203125, -0.2741149961948395, -0.002417516428977251, -0.015190365724265575, 2.112220525741577, 0.034844864159822464, 1.392976999282837, -0.9964472651481628, -0.37394583225250244, -1.03293776512146, 0.1326114684343338, -0.1403397023677826, 0.18586501479148865, 0.5819725394248962, -0.11790700256824493, -0.9487362504005432, -0.2201676219701767, -0.07117858529090881, 0.13566423952579498, 0.38936716318130493, 0.9897391200065613, -0.13824594020843506, 1.9498573541641235, 0.7145774364471436, 1.2092183828353882, 1.367142915725708, -0.17099304497241974, 1.1766859292984009, 0.5593791604042053, 0.5951055288314819, 0.55531245470047, -0.3579879403114319, 1.5205166339874268, 2.7576406002044678, -1.1360582113265991, -1.0844720602035522, -0.872322678565979, -0.33464792370796204, -1.1761342287063599, 1.6199402809143066, -0.646628737449646, -0.5642049312591553, -0.7098937034606934, -1.6786748170852661, 1.91614830493927, -0.7748063206672668, 0.02224140614271164, 0.39136362075805664, -0.34124770760536194, 1.1110666990280151, -0.630172610282898, -0.813854992389679, -0.16175998747348785, 0.5635581016540527, -0.2317514717578888, -0.40917906165122986, 1.4565774202346802, -0.20560328662395477, 0.5002855658531189, -0.5574907660484314, 0.10561010986566544, 0.08845045417547226, 0.7969086170196533, 0.5085499882698059, 0.1166481003165245, 1.2551348209381104, -0.29789888858795166, -0.8565331697463989, 0.894995391368866, 0.05254960060119629, 0.945690929889679, -0.22914627194404602, -0.27213019132614136, 0.5375925302505493, 0.1760753095149994, -1.3829907178878784, 0.35675346851348877, 1.1105808019638062, -1.2649949789047241, 0.7640631198883057, -0.7753177881240845, 0.2681683301925659, 1.5110571384429932, 0.538837194442749, 1.4256409406661987, -0.693760871887207, -0.4191925823688507, 0.27945926785469055, -0.5899516940116882, 0.0857163816690445, -0.8485438823699951, 0.2345154583454132, 1.0323925018310547, -0.7830655574798584, -0.16250616312026978, -1.0023667812347412, -0.5572307705879211, 0.4937707185745239, 0.34665295481681824, -0.7455806136131287, 0.27294406294822693, -0.4608429968357086, 0.7215880155563354, 0.12199120223522186, 0.28873956203460693, 1.047023892402649, -0.7345213890075684, 1.1883004903793335, 0.23298633098602295, -0.8105087876319885, 0.09632068872451782, -1.1463664770126343, -0.6836847066879272, -1.3659300804138184, 1.1821047067642212, 0.6572018265724182, 0.6162315011024475, 1.21951162815094, 0.4883577227592468, -0.2773679792881012, 0.7381356358528137, -1.0768324136734009, 0.6218093037605286, 1.9199751615524292, -1.6991162300109863, 0.7255235314369202, 1.2841384410858154, 0.882360577583313, -0.13080629706382751, -0.9048991203308105, 1.1940536499023438, 1.6445637941360474, -2.304033041000366, 0.5816665291786194, 0.1128741204738617, 0.6305757761001587, -0.8826507925987244, 0.19559264183044434, -0.4157753884792328, 0.3902537524700165, -1.1502329111099243, 0.6803563237190247, -0.6257784962654114, -0.8321322202682495, -0.7194294333457947, -1.1359689235687256, -0.42049095034599304, -0.1956702023744583, -0.1097879409790039, -1.8847658793674782e-05, 0.7522122263908386, -0.5151993036270142, -0.014981281012296677, -1.1507172584533691, -0.08322295546531677, 1.7289403676986694, 0.5921145677566528, 1.1195977926254272, -0.6675671935081482, -1.0337820053100586, -1.8678849935531616, -0.3628462255001068, 0.2669772207736969, -0.14933225512504578, 0.42545801401138306, -1.2958967685699463, -0.18758107721805573, -0.8300740718841553, -1.2558354139328003, -0.5811493396759033, 0.518841564655304, -0.48735731840133667, -1.653232216835022, 1.3112380504608154, -0.08612766861915588, 0.3253161907196045, 0.33685585856437683, 1.575508952140808, 1.5777291059494019, 0.04410398378968239, -1.7311079502105713, -1.2696641683578491, 1.3325731754302979, -0.18967311084270477, 0.3893604278564453, 1.3996573686599731, -0.1384836584329605, -0.8224433064460754, 0.45912283658981323, 0.16658245027065277, 0.5222840309143066, -1.0438746213912964, -0.47184616327285767, -0.07819206267595291, -1.6035091876983643, -0.4834633469581604, 0.06901852786540985, 0.3785818815231323, -0.7405716180801392, -0.6774740815162659, -1.1405106782913208, -1.088976263999939, 0.20949064195156097, -0.027446869760751724, -0.7372843027114868, -0.3623396158218384, 0.8998810052871704, -1.1643627882003784, -0.37198761105537415, 1.9868751764297485, 0.7287260890007019, -0.5811073780059814, 1.260299563407898, 1.1707535982131958, 0.25442224740982056, 1.5788393020629883, -0.644202709197998, 0.003900169162079692, -0.23303768038749695, 1.166935920715332, 1.6224182844161987, 1.1056313514709473, 0.8083582520484924, 0.8131541013717651, 1.144836187362671, -1.7220721244812012, -0.699157178401947, -1.2763500213623047, -0.030713019892573357, 1.0167944431304932, 0.3294677138328552, -0.538833737373352, 0.13522328436374664, -0.24056123197078705, -0.1077098697423935, 0.48967987298965454, 0.21937285363674164, -0.31504151225090027, -0.695777177810669, -1.1636712551116943, -1.782847285270691, -1.6580686569213867, 0.8849354386329651, 2.021883010864258, 0.4854070842266083, -1.0628600120544434, 0.6257258057594299, -0.5480756163597107, 1.0511871576309204, -0.16412144899368286, -2.3325231075286865, -0.2947578430175781, -1.291463851928711, -0.5391417741775513, -1.0400662422180176, -0.8870583176612854, -0.3525274097919464, -0.13091416656970978, 1.1560766696929932, 0.5213583707809448, 0.43050676584243774, 1.95644211769104, -1.1051018238067627, -0.10310366749763489, -0.21088646352291107, 0.1545064002275467, -0.3788451552391052, -0.2850664258003235, 0.8980470299720764, -1.0713621377944946, 0.46201762557029724, 1.5997458696365356, -0.43120718002319336, -1.1060490608215332, -0.25198304653167725, -0.4508764445781708, -0.4565922021865845, 0.47357815504074097, -2.075568437576294, -0.38599488139152527, 1.367519736289978, -0.21290987730026245, -0.8565711975097656, -0.6253927946090698, 0.952056884765625, 0.00054086500313133, -0.2305038869380951, 1.8989460468292236, 0.37639403343200684, 0.09601521492004395, 2.148441791534424, -1.7108824253082275, 1.18376886844635, -0.8099319338798523, 0.377922385931015, 0.1321054846048355, 1.8599202632904053, -1.4583247900009155, -0.44019997119903564, -0.5230602622032166, 0.23145778477191925, -1.2835246324539185, -0.6876919269561768, 0.2746003270149231, 0.5151455402374268, 0.9455969333648682, 1.519220232963562, -0.6651957631111145, -0.08292629569768906, -0.9190462231636047, 1.2289774417877197, 1.2593969106674194, -1.1759395599365234, 0.8411297798156738, -1.7519370317459106, -1.2453304529190063, 1.0535659790039062, -1.7107353210449219, 1.6309393644332886, -0.21223293244838715, -0.21162208914756775, -1.7290019989013672, 0.6607235670089722, -0.5365511775016785, 0.9760404825210571, 2.1862730979919434, -0.9934675097465515, -0.09264957904815674, -0.02591750957071781, 0.8537804484367371, -0.9586483240127563, 0.16009221971035004, 0.25682565569877625, -0.5797943472862244, 0.6409254670143127, 0.6086215972900391, 0.01855318434536457, -1.3406792879104614, -1.695542573928833, 1.282575011253357, 1.4993410110473633, -0.2069827914237976, 0.038893602788448334, 0.34030354022979736, 0.4526049494743347, 2.065761089324951, -0.021860603243112564, 1.8169745206832886, 0.6356187462806702, 0.08835356682538986, -2.1252057552337646, -0.3369360566139221, -1.1501296758651733, -1.0309709310531616, 0.490092009305954, -0.15069440007209778, -0.053724631667137146, -0.053120244294404984, 0.8966649770736694, 0.9085458517074585, 0.6391503214836121, -0.05306566134095192, -0.11996914446353912, 1.402032494544983, 1.8462705612182617, 1.7407886981964111, -0.19978192448616028, -0.12010478973388672, 0.7719765901565552, -1.3587673902511597, -0.4331737458705902, -1.050351619720459, -0.6715102791786194, -1.6134650707244873, -1.9393833875656128, -0.31003087759017944, -1.3385977745056152, 1.6236035823822021, -0.306275337934494, 0.37959808111190796, -0.013605082407593727, 0.8073347210884094, 0.9968653917312622, -0.3910740911960602, 2.462376594543457, -1.517433762550354, 0.06451211124658585, 0.12781062722206116, -1.0944383144378662, 0.08734413236379623, 1.3730714321136475, 0.29358410835266113, 1.8888887166976929, -1.0226290225982666, -0.1571800410747528, 0.5807765126228333, 0.8572481274604797, -1.9495688676834106, 0.2451871931552887, 1.8459659814834595, -0.6716399192810059, 1.2115724086761475, -0.7613635659217834, 1.085781455039978, -0.18004058301448822, -0.8161455988883972, 0.06976581364870071, 1.3245025873184204, 0.09064526855945587, 1.1169929504394531, 0.10519702732563019, -1.2072970867156982, 1.2168391942977905, 0.1381184160709381, -0.9417393207550049, -0.43388769030570984, -1.0460963249206543, 0.09683392941951752, 0.6101511120796204, -1.2544127702713013, 1.3736510276794434, -0.03647736459970474, -1.0256704092025757, -0.8062249422073364, -0.12280008941888809, -2.4452733993530273, -0.5831454992294312, -0.4681161344051361, 0.3397846519947052, -0.07634960860013962, 0.4682807922363281, 0.2592657208442688, 0.6473129987716675, -0.3409557640552521, 0.9730544090270996, 0.8042595982551575, 0.5818071961402893, -0.34264102578163147, -1.3129477500915527, -0.944323718547821, 1.331730842590332, 0.8009086847305298, -1.2658612728118896, -1.5387935638427734, -0.5764387249946594, 2.644263744354248, -1.2505629062652588, 1.7320111989974976, 1.7622836828231812, -0.5688554644584656, 0.6625726819038391, 1.5624361038208008, -0.10317377001047134, 1.770340919494629, -0.15209802985191345, -1.1263291835784912, 0.7558082938194275, -0.2229028046131134, 0.9099335074424744, -0.6047089099884033, 0.17194746434688568, 0.5707789659500122, 0.7948938012123108, 0.3728797137737274, 0.07290370017290115, 0.9015833735466003, 0.011802654713392258, 1.7370630502700806, 0.3437506854534149, 0.20655564963817596, 0.9962430596351624, -0.6076513528823853, 0.7457401752471924, 0.706136167049408, -1.3660566806793213, -0.3771646022796631, 1.5984220504760742, -0.7101674675941467, 1.9969675540924072, 0.4993937909603119, 0.54664546251297, 1.725688099861145, 0.009767130948603153, 0.17510740458965302, -0.24980837106704712, 1.4833018779754639, 0.11480820924043655, -0.6670132875442505, -0.33131030201911926, -0.6821736097335815, 0.795695424079895, -0.568469762802124, 0.46085482835769653, 1.5270049571990967, 1.5260441303253174, -1.8228137493133545, 0.35965588688850403, 1.0691149234771729, 1.2719308137893677, -0.9174875020980835, 0.2788007855415344, -0.9481331706047058, -0.26904577016830444, -1.5781422853469849, -1.4801074266433716, -2.0903143882751465, -1.6029067039489746, 0.06360528618097305, 1.0840904712677002, 0.3348817825317383, 1.9701980352401733, -0.4605672359466553, -0.6072531342506409, -0.10082835704088211, -1.0035021305084229, -1.3571385145187378, -0.9475412368774414, -0.7208912968635559, 0.9590843319892883, 0.2774137854576111, -0.39819619059562683, -1.951714277267456, -0.5135457515716553, -0.6975379586219788, 0.5127440094947815, -0.5860278606414795, 1.9634469747543335, 1.4857946634292603, 0.2898493707180023, -0.14105740189552307, -1.319962739944458, 0.33928653597831726, -0.3881849944591522, -1.1556059122085571, 0.8303510546684265, -0.6269297003746033, 0.2877432107925415, -1.022909164428711, -0.8040669560432434, -0.5500836372375488, 0.44129717350006104, -0.7614355087280273, -1.4599405527114868, -0.20720918476581573, 1.5162960290908813, -1.9611579179763794, -1.8931225538253784, 0.2881409227848053, 1.6552119255065918, -1.3622498512268066, -1.0692112445831299, -0.2774485945701599, 0.4581930339336395, 1.4257487058639526, 1.9538681507110596, 0.47750425338745117, -0.22823993861675262, -1.50882887840271, -0.7096756100654602, -0.5517669916152954, -1.102441668510437, 0.9150419235229492, -1.1184511184692383, -1.3067549467086792, -1.4396871328353882, -1.5026298761367798, 1.220035433769226, -1.3138455152511597, -0.6804459691047668, -2.035432815551758, 0.23069894313812256, 2.053499460220337, -0.2325311005115509, -0.8562401533126831, 0.007598617114126682, -0.012421022169291973, 0.8906503915786743, 0.4895460903644562, 1.2282202243804932, -0.6055177450180054, 0.36642593145370483, -1.0996254682540894, -0.04780277982354164, 0.11358638107776642, -1.251175045967102, 1.1628004312515259, -0.21446523070335388, -1.2203760147094727, 0.005365902092307806, 1.268620491027832, 1.2177292108535767, -0.6339212656021118, -0.16722074151039124, 0.3096141815185547, -0.44240862131118774, 1.43656325340271, -0.6785371899604797, -0.17409345507621765, -0.7393226027488708, 0.4341890513896942, 0.13826896250247955, -1.601524829864502, -2.088942289352417, 1.7793099880218506, -1.7358239889144897, 0.03413262218236923, -0.5673349499702454, 1.1860222816467285, -0.8004851937294006, 0.036429513245821, -0.42791077494621277, -0.6076475977897644, -1.043353796005249, 0.2233130782842636, -0.2796478867530823, -1.1576197147369385, -0.002996626775711775, 1.200842022895813, 0.8196439743041992, 1.9474502801895142, 0.7654650211334229, 0.5446376204490662, 0.3695266544818878, -1.3905506134033203, -0.9259217977523804, -0.36526891589164734, -0.8099931478500366, 0.5709061622619629, 2.9508419036865234, 1.0362378358840942, -1.952682375907898, 1.2260833978652954, -0.5519667863845825, 0.5475103259086609, -0.02583698183298111, -1.1397159099578857, 0.6001969575881958, 0.30283913016319275, 0.6528304219245911, -0.1274232566356659, 0.4258941113948822, 1.617605447769165, -1.3455005884170532, -0.5013144612312317, 0.5863159894943237, 0.2018042355775833, 0.15190844237804413, 0.27327558398246765, -0.33320096135139465, 1.3833839893341064, -0.8223105669021606, -0.7521100044250488, 0.07842801511287689, -0.8201906681060791, 1.7247556447982788, -2.6614279747009277, -0.45080021023750305, -0.5662928819656372, -1.2189197540283203, 0.16608090698719025, 0.698689877986908, -0.8623906970024109, 0.23345129191875458, -0.09261882305145264, 0.4325021505355835, 0.09539426863193512, 0.9349466562271118, 1.3809643983840942, 2.09279203414917, -0.918776273727417, -0.21758460998535156, -0.11312739551067352, 1.7770720720291138, 0.23042979836463928, 0.8505323529243469, 0.5109968185424805, -1.6496422290802002, 0.32377415895462036, -0.602832019329071, -0.4701641798019409, 0.20873622596263885, -0.5225575566291809, 0.9779903888702393, 1.0731449127197266, -0.9029017090797424, 2.03711199760437, -0.5222962498664856, 1.128764033317566, 1.1599047183990479, -1.361448049545288, -0.5509979724884033, -0.7037550210952759, -0.27040886878967285, 0.38493114709854126, 0.8606027364730835, 1.407552719116211, -0.235494926571846, -1.4898381233215332, 1.4545481204986572, 0.6346235275268555, -0.980434775352478, 0.2577213943004608, 0.11710771173238754, 1.1288820505142212, 1.13687264919281, -0.4350360333919525, 0.8706746101379395, 1.4986188411712646, 0.8090082406997681, 1.4270555973052979, 0.8696872591972351, -2.0712764263153076, 0.014630179852247238, 0.09263864904642105, 0.23902851343154907, 0.7784809470176697, 0.46001046895980835, 0.5942183136940002, 0.8282901048660278, 1.0604274272918701, 0.7019805312156677, -1.0061089992523193, 1.2126954793930054, -0.3031121790409088, -0.009906894527375698, -0.6362451314926147, -0.6497741937637329, -0.3119172751903534, -1.3938175439834595, -0.3714264929294586, 0.4878941476345062, -0.4189971387386322, -0.4054747223854065, -0.5160502195358276, 0.8502368927001953, 0.8185058236122131, -0.28524285554885864, 0.5672953724861145, -0.07431857287883759, 1.4420433044433594, 0.7488492727279663, -1.0394952297210693, 0.6826720833778381, -0.7174988389015198, 0.06573627144098282, -1.8482414484024048, 0.39364686608314514, -1.1832014322280884, -0.30454590916633606, -1.369057059288025, 0.20582430064678192, -0.6992347836494446, -0.03164517506957054, 0.5748726725578308, 0.1581452190876007, 2.274757146835327, -1.4729982614517212, -1.4609726667404175, 0.157956063747406, -0.3285786807537079, 1.4473819732666016, 0.3328859508037567, -0.28139349818229675, 0.3431456685066223, -0.33785519003868103, -1.5849294662475586, 0.6497657895088196, -1.5237715244293213, 0.7575492262840271, -1.773332953453064, -0.7938652038574219, -0.12424622476100922, -0.43909916281700134, -0.11141885817050934, -1.0948195457458496, -2.1183571815490723, 1.9290878772735596, 0.4156012535095215, -0.6701850891113281, 0.7804708480834961, -0.38519978523254395, -0.026102876290678978, -0.4684341847896576, 1.4924181699752808, -1.1997127532958984, 0.7096508145332336, -1.6338328123092651, 0.16436296701431274, 1.3096344470977783, -1.1168991327285767, -0.19394877552986145, -0.7397366166114807, 0.3879784345626831, -0.6082841157913208, 0.6500323414802551, -1.19076669216156, -0.04134676977992058, 0.9198186993598938, -1.1042990684509277, -0.6252121925354004, -0.4117490351200104, 1.2402232885360718, 0.12861095368862152, -0.6580348014831543, 0.011053265072405338, 0.484988272190094, 1.6994727849960327, 1.8697279691696167, 2.225825309753418, 0.6837860941886902, -0.0565946102142334, -1.5788110494613647, 1.031195878982544, -0.9408618807792664, -0.6607587933540344, -0.6545715928077698, -0.06807595491409302, -0.13005387783050537, -0.38653433322906494, -1.3919336795806885, -0.5101706981658936, 1.1246416568756104, 0.48296236991882324, -1.9195791482925415, -0.17481786012649536, 0.08379548043012619, 0.7469152808189392, 1.1028058528900146, 0.70392906665802, 1.5564906597137451, -0.3710263967514038, -0.5028008818626404, -1.9776606559753418, 1.9539527893066406, 1.7049264907836914, -1.371362328529358, -2.001777410507202, 0.008674458600580692, -0.4095555543899536, 0.1416737288236618, -0.7138634324073792, 0.555303692817688, -0.8157246708869934, 1.3752593994140625, 1.144327163696289, 1.6667628288269043, -0.3976075053215027, -0.2288840413093567, 0.11803209781646729, 0.6445693373680115, 0.09291724860668182, 0.38619574904441833, 0.78011554479599, -1.702319622039795, 1.2438234090805054, 1.5449284315109253, -1.8007569313049316, -0.9866524934768677, 0.44015833735466003, 1.2856230735778809, 0.944665253162384, 0.15235044062137604, -0.1718175858259201, 0.35517680644989014, -1.4756196737289429, -0.3548455238342285, 1.5778930187225342, -1.6369128227233887, 0.297842800617218, -0.4504188299179077, -2.5130722522735596, -0.14601318538188934, 0.6640787720680237, 0.6960991024971008, 0.3035120964050293, -0.22713439166545868, -0.8359264135360718, 1.0855857133865356, -1.0930821895599365, -0.48338523507118225, -0.26877647638320923, -0.770635724067688, 0.6678245663642883, -0.4426763355731964, -1.5576434135437012, -1.5130022764205933, 1.9126007556915283, -0.8947705626487732, 1.3990892171859741, -1.3006188869476318, 1.6131747961044312, -1.2965707778930664, -0.5676567554473877, 0.5756366848945618, -0.1114133968949318, -1.616140365600586, 0.016048384830355644, 1.3730227947235107, 0.01554662361741066, -0.07028191536664963, 1.5815261602401733, -0.18175435066223145, -0.04192911833524704, 1.4327986240386963, -0.8557531833648682, -0.4155113697052002, 1.1792054176330566, -1.0917755365371704, 0.7660345435142517, 0.5893987417221069, 0.29778745770454407, 0.019808895885944366, 1.0066726207733154, 0.5727275609970093, -0.942712128162384, 1.2287009954452515, -0.4620932638645172, 0.572173535823822, -0.400477796792984, -0.07784535735845566, 0.7883405685424805, 0.24653983116149902, 0.011794650927186012, 0.13525643944740295, 1.0117590427398682, 0.2907240390777588, 0.4119253158569336, 1.85769784450531, -0.3187984824180603, -0.293634831905365, 0.7314751744270325, 0.18783092498779297, -1.8684402704238892, 0.488789826631546, 1.3632808923721313, 1.4905015230178833, -0.01728592813014984, -0.722743034362793, 0.18850260972976685, 1.1837748289108276, -1.2165513038635254, -2.7278685569763184, -0.9807906746864319, -0.058676015585660934, 0.9496213793754578, 1.0021533966064453, -0.9562498927116394, 0.06937728822231293, 0.778374195098877, 0.3465784788131714, 0.25385233759880066, -1.120514988899231, 0.46711117029190063, -0.9923709630966187, -0.07924901694059372, 0.39105239510536194, -0.9749622344970703, 1.3741759061813354, 0.22588957846164703, -0.44825661182403564, 2.4333226680755615, -0.5386723279953003, -0.6273688673973083, -0.08833429217338562, -1.7775169610977173, -0.5504086017608643, 0.4612070620059967, 0.6592003107070923, 0.6891653537750244, -0.9155827760696411, -1.2657850980758667, 0.7197169065475464, 1.4551395177841187, 2.5006134510040283, -0.5578492283821106, 0.515180766582489, -0.18961158394813538, -2.0050153732299805, 0.7296176552772522, -0.7766984105110168, -0.6934176087379456, 1.024896264076233, 0.9710820913314819, -1.149901270866394, 0.24654437601566315, 0.5060843229293823, -1.4357357025146484, 1.6113543510437012, 0.5732501149177551, 0.9241198301315308, -0.2234753519296646, -0.501043975353241, 1.076195240020752, 1.2562098503112793, -1.6815763711929321, -0.6275820732116699, -0.37283629179000854, 0.32536256313323975, -1.0124149322509766, 0.6408810615539551, 3.0660934448242188, 0.9749757051467896, 0.9100260138511658, -0.3025703728199005, 0.20921112596988678, -1.3488520383834839, 0.41709065437316895, 1.9329679012298584, 0.601068377494812, 0.9022808074951172, 0.6199178695678711, -0.3820343017578125, 0.7101209759712219, -0.06209303438663483, -0.15103581547737122, -1.1063623428344727, 1.7842341661453247, -0.9364523887634277, -0.947590172290802, 1.5708860158920288, -0.005034332629293203, -0.398146390914917, -0.11900164932012558, 1.297466516494751, -0.7446805834770203, 2.7681026458740234, 0.5784987211227417, -0.5089004635810852, 1.0628960132598877, -0.5009397268295288, 2.2046144008636475, 0.2855919897556305, -0.783189594745636, 0.17319531738758087, 0.7476520538330078, 0.2689528167247772, -0.7012209892272949, 0.12371517717838287, 1.107806921005249, 1.1098960638046265, -0.5905064940452576, 0.7057124376296997, -0.35127905011177063, 1.285496711730957, -2.090097188949585, -0.8398289680480957, -0.4382416307926178, -0.7832157611846924, -0.16439957916736603, 0.978387713432312, 0.058271732181310654, -0.7329452037811279, -0.15417972207069397, -0.3899422287940979, 0.5773377418518066, -0.47150135040283203, -0.1465306431055069, 0.7212669253349304, 1.4143487215042114, 0.255311518907547, -0.3755844235420227, -0.8151174783706665, 1.2598795890808105, -0.7516022324562073, 0.07099822163581848, 1.4823211431503296, 0.14038823544979095, 0.23437610268592834, 0.799150824546814, 1.1456955671310425, 0.8939585089683533, 0.6616023778915405, 2.2066664695739746, 1.3490301370620728, 0.3867020606994629, 0.3105432689189911, 0.40117979049682617, 0.9697721004486084, -1.8041547536849976, -0.3951702415943146, 0.10642405599355698, 1.440949559211731, -0.12701515853405, -0.04978662729263306, -1.220970869064331, 1.236770749092102, 0.9125277400016785, -1.2819067239761353, 0.45769429206848145, 0.8254850506782532, -0.45648062229156494, 1.9202885627746582, 0.07346225529909134, -0.12817391753196716, 0.14331801235675812, -1.1381115913391113, -0.3594534695148468, 1.1201002597808838, 0.09660819917917252, 0.7924465537071228, 1.403077244758606, 0.6110184192657471, 0.20294071733951569, -1.7066363096237183, 1.2779598236083984, -0.7961759567260742, -0.22141461074352264, 0.8103823065757751, 0.22135034203529358, -0.4602406322956085, 0.6505894064903259, -0.2194090187549591, -1.2589640617370605, -0.0420999713242054, 0.0072114416398108006, 0.029417430981993675, 0.27778223156929016, -0.5873515605926514, -0.3711370527744293, 0.41204172372817993, 0.6901347637176514, 1.2013208866119385, -1.1783298254013062, 0.030512461438775063, 0.12873829901218414, 0.6438804268836975, -1.3622636795043945, 0.8044654130935669, -0.5357439517974854, -0.021636594086885452, -2.394243001937866, 0.8192092776298523, -0.2543962597846985, 0.6230857968330383, 2.198376178741455, -1.6053080558776855, 0.34672731161117554, -1.4879631996154785, 0.14455389976501465, -0.4637918174266815, -0.8812130093574524, -0.8591528534889221, -0.39091262221336365, -1.8843905925750732, -0.6757696866989136, 0.44519829750061035, -1.1526215076446533, -1.1890485286712646, 0.4453412890434265, -0.785824179649353, 0.6698769927024841, -0.48433682322502136, -0.8164300322532654, 0.14338034391403198, -0.6993908882141113, -0.060681331902742386, 1.0301449298858643, 2.3186709880828857, 0.6136144995689392, 0.03349016234278679, 1.005646824836731, 0.3185383081436157, -1.1509554386138916, 1.2094217538833618, 0.5452023148536682, -0.0754772499203682, 0.4158807098865509, -1.1453330516815186, 1.4134200811386108, 0.32452136278152466, 0.5322546362876892, 0.8541932106018066, 0.046943530440330505, 1.0816222429275513, -0.20289112627506256, 1.1223957538604736, 0.9604119658470154, 1.7755999565124512, -1.4004791975021362, -0.30649346113204956, 1.8171411752700806, 0.48002657294273376, -2.636648416519165, 0.8009931445121765, -1.1119308471679688, 1.5634816884994507, -0.6554456949234009, 3.170358657836914, -0.012850173749029636, -0.34243160486221313, -0.308313250541687, -0.1413663923740387, -0.58232581615448, -0.6539773941040039, 0.15536247193813324, -0.08448400348424911, -1.9143850803375244, -0.5112048983573914, -0.7633864879608154, 0.4586748778820038, 0.8668140172958374, -0.021131055429577827, 1.9240049123764038, 0.18785445392131805, 0.8891956806182861, 0.3010774850845337, -1.7737170457839966, -0.6174716949462891, 0.3825080990791321, -1.4484782218933105, 1.1029549837112427, 0.7991090416908264, -0.2731205224990845, 1.215352177619934, -0.5471682548522949, -1.2995353937149048, -1.1170809268951416, -1.0404924154281616, -0.6626980304718018, 0.5902515053749084, 2.0097250938415527, -2.537773609161377, -1.5134410858154297, 1.044541358947754, 0.19865810871124268, 2.240687847137451, -0.4800567030906677, 1.364704966545105, -0.04420074447989464, 0.35592979192733765, -0.013396037742495537, 0.009250680916011333, 2.8977267742156982, 1.2667644023895264, -1.138597846031189, -0.6712172031402588, 0.3691169321537018, 1.9641095399856567, -0.22607846558094025, -1.4867675304412842, -0.9325985908508301, -0.31512507796287537, 0.7862627506256104, 1.4519999027252197, -1.5009101629257202, 0.5691066980361938, -0.18843109905719757, -0.08726540952920914, -0.12003482133150101, 0.2253178060054779, 0.5823936462402344, 0.14328637719154358, 1.1808323860168457, 0.6994128823280334, -0.9115381240844727, 1.6897642612457275, 0.6329337358474731, 1.9117716550827026, -1.7654188871383667, 0.6237030029296875, -0.6442907452583313, -0.07409632951021194, -0.02628679759800434, 2.2479653358459473, -0.052924204617738724, 2.3036468029022217, -2.7922935485839844, 0.7040112614631653, 0.15179140865802765, 0.4308401644229889, 0.8432902693748474, -1.4591153860092163, -0.05999533087015152, -0.6048274636268616, -0.19314232468605042, -0.4473264217376709, -1.5251798629760742, -2.2818658351898193, 0.16929131746292114, -0.8482480645179749, -0.1689348667860031, -0.013784854672849178, -1.1897281408309937, 0.755558431148529, 0.7737674117088318, 0.6028295755386353, 0.5386959910392761, 1.0689057111740112, -1.0165852308273315, 0.7575514316558838, 0.8569669723510742, 1.364456295967102, 0.5580812692642212, -0.5246194005012512, 0.2162373661994934, -0.5719372034072876, 0.8429656028747559, -0.8933548927307129, -0.20155148208141327, -0.008696100674569607, -0.9113970994949341, -1.5454144477844238, -0.10963943600654602, 0.15991926193237305, -0.45983627438545227, 1.2932020425796509, 1.388258695602417, 1.2762361764907837, 0.30566829442977905, 1.1277607679367065, 1.0399926900863647, -0.40468353033065796, 0.6607378721237183, 1.1385879516601562, -0.3541634976863861, -1.2301175594329834, 0.20244160294532776, 1.1494613885879517, -0.05630558729171753, 1.1646407842636108, 0.4665622413158417, -0.28833308815956116, 0.15245412290096283, 0.2125004380941391, 0.2789161205291748, -0.4958971440792084, -0.8582355976104736, 0.8975745439529419, 0.2687954008579254, -0.3351154625415802, -1.4865142107009888, -0.6390364170074463, -0.05597148463129997, -1.0213780403137207, -0.7962063550949097, -0.3707766532897949, -2.1846961975097656, -1.1506375074386597, 0.15747973322868347, -2.24989652633667, -1.4508756399154663, -1.7153595685958862, 0.9316906929016113, -1.3125505447387695, 0.08363451808691025, -0.7852234244346619, -0.12532494962215424, -0.6026338934898376, 0.4808080196380615, -1.2446866035461426, 1.7346446514129639, 0.6283039450645447, 0.1257830560207367, -0.7082250714302063, 1.4472033977508545, -2.217215061187744, -0.08877421915531158, 0.8657022714614868, -0.17969952523708344, -0.6690601110458374, -1.9835842847824097, 0.7537633776664734, 0.6135523319244385, -0.9637092351913452, 1.8901110887527466, 0.4577060639858246, -0.6325873732566833, -0.11249957233667374, -0.10718300193548203, 0.5966426134109497, -0.41214582324028015, 1.0102051496505737, -0.9211956262588501, 2.14420223236084, -0.5725297927856445, 0.9634842276573181, 1.804824709892273, 1.1220396757125854, 0.36298006772994995, 1.2991886138916016, -0.5157289505004883, -1.2731366157531738, 0.8656793832778931, -0.15967102348804474, 0.06349142640829086, 0.4725593328475952, -0.6688099503517151, -0.15062178671360016, -0.036581214517354965, -0.11073154956102371, 0.48173707723617554, -0.04673825949430466, 0.443111777305603, -0.884548008441925, 0.22236572206020355, -1.2192296981811523, -0.5297422409057617, -0.05821976065635681, 0.08944641053676605, -0.8722453117370605, 0.924811065196991, -0.16924208402633667, 0.8790608048439026, -0.6739726066589355, 0.24497073888778687, -0.5468860268592834, 1.7259231805801392, -0.20894283056259155, -1.1713776588439941, -0.32050400972366333, -0.19086329638957977, -0.1688501238822937, -0.20213273167610168, -0.5419114232063293, 0.7450330257415771, 0.5812878012657166, -0.7923881411552429, -0.5042446255683899, -2.402533769607544, -2.1367101669311523, 1.5841811895370483, 1.4391437768936157, 0.6738386750221252, -2.216965436935425, -1.0472190380096436, -0.9795429110527039, -0.9942536354064941, -0.6532489061355591, -0.5573933124542236, -0.034047726541757584, 0.07856069505214691, 1.4172906875610352, -1.1599539518356323, -0.5968075394630432, -0.1988578587770462, -0.37237581610679626, -0.33685481548309326, 0.1501404196023941, 1.7390440702438354, 0.5784443020820618, -0.8755105137825012, -1.299402117729187, -0.44066452980041504, -0.46309059858322144, 0.7141185998916626, -0.06006557494401932, -1.8916666507720947, -0.7141631245613098, 0.24806445837020874, 0.7444069981575012, 0.11721885949373245, 0.023360829800367355, 0.17824235558509827, 0.7058266401290894, 0.6814275979995728, 1.2391935586929321, 0.3657185733318329, -0.1836436241865158, 1.455128788948059, -0.03803861513733864, -2.072028875350952, -1.3995270729064941, 0.7282345294952393, -0.29571783542633057, 1.1909948587417603, -0.26120254397392273, 1.1723535060882568, -1.706228256225586, -0.09663903713226318, 1.7072346210479736, -0.7548621296882629, 0.12017618864774704, -1.107161283493042, -1.0866940021514893, 1.5107804536819458, 0.3993813693523407, -1.2509846687316895, -0.8118950128555298, 0.07237987220287323, -0.46019527316093445, -1.0274741649627686, -1.5772329568862915, -0.6881378293037415, -1.1817283630371094, 1.1779499053955078, 0.16199201345443726, 1.4109901189804077, 0.7502924203872681, -0.9717003703117371, -1.1462182998657227, 0.7081499099731445, 0.26258864998817444, 0.7518513798713684, 0.562879204750061, 0.8379868865013123, -0.637874186038971, 1.3575481176376343, 1.178561806678772, 0.33386021852493286, 0.8826302886009216, 0.9661771059036255, -1.295868992805481, -1.494533896446228, -0.6307963132858276, 0.3568885326385498, 0.7243652939796448, 0.28014346957206726, 1.6049903631210327, 0.6174285411834717, 1.6338000297546387, 0.9704845547676086, 0.9906306266784668, 0.5025238394737244, -1.3700271844863892, -0.6303232908248901, 0.5372734665870667, -1.8844937086105347, -0.8992665410041809, -0.5123537182807922, -0.8508500456809998, -0.01095679122954607, -0.5935937166213989, -0.06549147516489029, -0.7755470871925354, 0.3513989746570587, -2.292370080947876, -0.39998292922973633, -0.012069791555404663, 2.005554676055908, -1.745413064956665, 0.5739617347717285, -0.6806972026824951, -1.2174137830734253, 2.014785051345825, 0.3554113507270813, 1.6282957792282104, -1.1886569261550903, 0.05738583207130432, -0.5697619318962097, -0.9893890619277954, -2.234889268875122, 0.3329828977584839, 1.013073205947876, 2.2861928939819336, -0.6591397523880005, 1.1256202459335327, 1.3309794664382935, 1.868175983428955, -1.0505543947219849, 0.2248971164226532, 1.0921094417572021, 1.6011126041412354, -0.7943795323371887, 0.7598776817321777, -1.6285964250564575, -0.811477780342102, -0.12886157631874084, -1.64358389377594, 0.11632313579320908, -0.0897437334060669, 3.418867349624634, 0.5307906866073608, -0.5895872712135315, 0.13324901461601257, -1.3059558868408203, -0.35309624671936035, -0.5014845728874207, -1.364129662513733, 0.19517377018928528, 0.7197655439376831, 0.6224006414413452, -0.655117928981781, -1.2377692461013794, -0.9342034459114075, 1.5739405155181885, 1.0027137994766235, 0.4685201942920685, 0.8442180156707764, 1.1214873790740967, -0.45233142375946045, 0.7179957032203674, 0.0338750034570694, -0.12613680958747864, 1.5437442064285278, -0.7984792590141296, 0.4368647336959839, 1.0311973094940186, 0.7629926800727844, -1.5814357995986938, -0.05910423770546913, -0.9987607598304749, 0.16605103015899658, -0.10500086843967438, -1.076140284538269, -0.15291756391525269, 0.4180000424385071, -0.7301961779594421, 0.17510156333446503, 0.19853928685188293, -0.11831191182136536, -2.49780011177063, -1.5641489028930664, -0.43049776554107666, 0.7165066599845886, 0.5299323201179504, -0.5320138335227966, -0.09168318659067154, -0.644870936870575, -1.2379891872406006, 2.8509409427642822, 0.8401954174041748, -0.5807979106903076, -1.0398653745651245, -0.20239710807800293, 1.7179726362228394, 0.0792464166879654, -1.3093981742858887, -1.8522067070007324, 0.731664776802063, -1.2668787240982056, -0.749254584312439, -0.22995513677597046, 0.16210436820983887, 0.28881898522377014, -0.10170324146747589, -0.4074162244796753, 0.9666532278060913, 0.9985283017158508, -0.4806390702724457, -0.44220101833343506, -0.5394516587257385, -1.1819158792495728, 0.904075026512146, 0.02686404436826706, 2.1989309787750244, -0.5297955274581909, -0.6452092528343201, -3.2366573810577393, 0.29212453961372375, 0.3134971559047699, 1.3062410354614258, 1.3507293462753296, 1.365992784500122, -0.1559128314256668, 1.3588099479675293, -1.0862915515899658, -1.9248119592666626, -0.6281979084014893, -0.2721918523311615, 0.8312883377075195, -2.202514886856079, -1.40865957736969, -1.9125049114227295, -2.4101011753082275, 0.8041374683380127, -0.27812936902046204, 1.538588285446167, -0.34235838055610657, -0.7356865406036377, -0.219509556889534, -0.4583311378955841, -1.3131519556045532, -0.9235681891441345, 0.3249853551387787, -0.7521902918815613, 0.33384501934051514, -1.8569765090942383, -1.3245950937271118, 1.334373116493225, -0.4930991530418396, 0.30593565106391907, -1.3450472354888916, -0.6464681625366211, 1.6545969247817993, -0.045597273856401443, -1.293116569519043, -0.1527729034423828, 0.9163162112236023, -0.07739990204572678, -0.07948268949985504, -0.5218825936317444, 0.39753100275993347, -0.2900192439556122, 0.9754095673561096, -0.8410808444023132, -0.2541152834892273, -1.2150113582611084, 0.8812957406044006, 1.1155495643615723, 0.15097284317016602, -1.1447196006774902, 0.7874465584754944, 0.8756666779518127, -0.8879587054252625, 1.2509076595306396, -0.1155766174197197, 0.7487856149673462, 0.4544467628002167, -0.5821667313575745, 1.7288511991500854, -0.3997000753879547, -0.07843223959207535, -0.6266594529151917, 0.6480123996734619, 1.4373071193695068, -0.07583258301019669, -0.4411911070346832, 0.7189557552337646, -0.6912691593170166, -1.5275059938430786, -1.6702115535736084, 1.3042271137237549, -0.216547891497612, 0.6034144759178162, 0.2849389314651489, 0.7553295493125916, 0.35431140661239624, -1.9592576026916504, 1.256406307220459, 1.1732667684555054, -0.5699992179870605, -0.1400369107723236, -1.0920262336730957, -0.6652234792709351, -1.004878044128418, -0.8556952476501465, 1.2806854248046875, 0.15423928201198578, -0.5257905125617981, 0.6365174651145935, -1.5387253761291504, 1.610068678855896, -1.1012771129608154, 0.3050972819328308, 0.4010998010635376, -1.452995777130127, -1.679525375366211, -0.25301724672317505, 0.16666333377361298, 1.2013417482376099, -0.44711562991142273, 0.2932499051094055, -2.9586293697357178, 1.1475834846496582, 1.7550429105758667, 0.3126107156276703, -0.10216906666755676, -0.3837672770023346, 1.6997102499008179, -0.5286605954170227, 0.09682050347328186, -0.14805454015731812, 1.8887243270874023, 0.8569326400756836, 2.567192554473877, -0.35298776626586914, -0.8064135909080505, 0.26760461926460266, 0.4162467122077942, 0.10571498423814774, 0.4381888210773468, 0.9437010884284973, -0.17685841023921967, 0.841046154499054, 1.369953989982605, -0.5748928189277649, -0.3930304944515228, -1.2106186151504517, -0.6856237053871155, -0.3525141477584839, -0.3785863220691681, -2.8409101963043213, 0.03606542572379112, 0.7831980586051941, -1.5949556827545166, -0.03378269076347351, 0.33504340052604675, 0.6422979235649109, 2.4862146377563477, -0.5772619247436523, -2.209322929382324, 0.5854365825653076, 0.8177801370620728, 0.8457033038139343, 0.3696243166923523, -0.7575794458389282, 0.6526201367378235, -0.17281474173069, 0.37717509269714355, -0.2884722054004669, 0.6094555854797363, 0.3300538957118988, -0.057503025978803635, 1.000659704208374, -0.5825062990188599, -0.39625364542007446, -1.1920115947723389, 1.1670703887939453, -0.9380910396575928, -0.4320567846298218, -1.1368976831436157, 1.2642267942428589, 1.812796711921692, 0.6937856078147888, -1.7117059230804443, -0.6269944906234741, -0.9384108185768127, -0.8389921188354492, -0.7915489077568054, -0.07168322056531906, 0.6010386347770691, 1.054002285003662, 0.13262401521205902, -0.40483149886131287, 0.867018461227417, -0.4530477225780487, 0.5198054313659668, 1.1012991666793823, -0.16204683482646942, 0.6195911169052124, 0.9811193943023682, 0.3822384476661682, -0.9578600525856018, -2.043658494949341, -1.0088589191436768, 1.1769567728042603, 0.554171621799469, 0.02794041857123375, -0.7249773144721985, 1.618634581565857, 0.213203564286232, -0.06469760090112686, 0.064895398914814, -0.5059561729431152, 0.5402395129203796, 1.0902211666107178, 0.6467969417572021, 3.2347676753997803, 0.10097939521074295, -0.5685110688209534, -0.581704318523407, -2.5370030403137207, -1.5925301313400269, 0.5000801682472229, 0.5926878452301025, 0.7373540997505188, -0.048814497888088226, -0.18185384571552277, -0.06144654005765915, 0.4980085790157318, 0.06920760869979858, -0.9959399700164795, 0.908479630947113, -1.3809717893600464, -0.003219794249162078, 0.7266541123390198, 0.5716092586517334, -0.5123236179351807, -0.695022702217102, 0.32239389419555664, 1.0421830415725708, -0.8445259928703308, -0.4872085452079773, -0.028663400560617447], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[-1.3278875350952148, 1.1860384941101074, -0.002977785188704729, 0.9836180210113525, 1.311061978340149, -1.0240635871887207, 1.084900140762329, -0.6240049600601196, 0.035344403237104416, -0.0798833817243576, -0.24597211182117462, -0.9112240076065063, -0.6717326045036316, -0.1405671387910843, -0.4039572775363922, 0.1837095469236374, 0.9298658967018127, -1.4470605850219727, -0.5002302527427673, 0.04586139693856239, 0.431194931268692, -0.08679963648319244, 0.0665302649140358, -0.29484784603118896, 0.9120671153068542, -0.5871638059616089, -0.3605574369430542, -0.06636898219585419, 4.3610053062438965, -0.25201672315597534, 0.5106995701789856, -0.9302306771278381, 0.21482409536838531, 0.007771060802042484, -1.4058067798614502, -1.0895593166351318, 0.27834436297416687, 0.27752599120140076, -0.05584948882460594, 0.37359943985939026, 0.5801424980163574, -0.30897635221481323, 2.3990280628204346, -0.09409832954406738, -1.4414628744125366, 0.3698470890522003, -0.2689298391342163, -0.6338602304458618, -0.3839142322540283, 0.6176621913909912, 1.4912770986557007, -2.2171828746795654, 1.4649593830108643, 3.068965435028076, 0.008740279823541641, -1.1961339712142944, -2.3290791511535645, -0.0077392105013132095, -0.12624025344848633, 0.5963897705078125, -0.7602705359458923, 1.0861154794692993, 0.19850055873394012, -0.4064975380897522, 0.09984666854143143, 0.00763540668413043, 0.06510723382234573, 0.18338261544704437, -0.5209335088729858, -0.35714420676231384, 0.2009553462266922, -1.4486888647079468, -0.8800662159919739, -1.319514513015747, 0.5801573991775513, 0.5277146697044373, -0.04970931634306908, 0.1474347859621048, 0.15047551691532135, 0.017913727089762688, -2.802957773208618, 0.6315789222717285, 0.9655013680458069, 1.7948740720748901, 2.6862339973449707, 0.029129652306437492, -0.16061539947986603, -0.23622281849384308, 1.2037545442581177, 0.14243346452713013, -0.04990943521261215, 0.4336646497249603, -0.058536648750305176, -0.014031183905899525, -0.18623541295528412, 0.6407245397567749, -0.8389163017272949, -0.15306958556175232, -0.906134307384491, 0.0019170107552781701, 0.21118949353694916, 0.8823021054267883, 0.8433621525764465, 1.7867344617843628, -0.3430822789669037, 0.03212148696184158, -1.299336314201355, -1.952649712562561, 0.009328431449830532, -0.48216745257377625, -0.3155604898929596, 1.48972487449646, -0.517175018787384, -0.029230743646621704, 0.6610361337661743, 0.4158205986022949, -0.0040539586916565895, -0.26385602355003357, -0.23180900514125824, -0.38128462433815, 1.233443260192871, -0.16582290828227997, 0.05614464730024338, -0.14332976937294006, -3.328495502471924, 0.016308482736349106, -0.3848468065261841, -0.2431713342666626, 0.05124404653906822, -0.16976206004619598, -0.16382844746112823, 0.02849043346941471, 0.06301162391901016, 0.31393539905548096, -0.1479211449623108, -0.6325919032096863, 0.0008813031017780304, 0.2909083962440491, -0.032489113509655, 0.14188936352729797, 0.067334845662117, 0.10788683593273163, -0.06760276854038239, 1.8631595373153687, -1.6841195821762085, -0.18083392083644867, 0.5458435416221619, -1.2893263101577759, -0.13492706418037415, -1.979990005493164, 2.0427405834198, -0.42831817269325256, -0.04799092188477516, -0.6402396559715271, -2.8275864124298096, 2.0122671127319336, 0.1792253702878952, -1.1439182758331299, -1.7107295989990234, -1.0566655397415161, 0.06820186972618103, 0.12714335322380066, -0.021785715594887733, -1.0874016284942627, -0.14113779366016388, -0.9257730841636658, -0.046855226159095764, -0.16352036595344543, 1.4841450452804565, 0.5384070873260498, -1.291176676750183, -0.29211539030075073, 2.4665367603302, -0.13839109241962433, 3.549443483352661, -0.1120181530714035, -0.14145442843437195, -0.35287943482398987, 1.2628785371780396, -0.4930937886238098, -0.39085853099823, 1.227156639099121, -0.2498311698436737, 0.38560056686401367, 1.7654650211334229, 0.1442871391773224, -0.0003813846269622445, -0.024730579927563667, -3.8994593620300293, -0.00383674050681293, -0.1682816445827484, -0.19786441326141357, 0.10091204941272736, 0.20660538971424103, 0.19958029687404633, -0.15240131318569183, 0.037985362112522125, 0.22191879153251648, 0.10983235388994217, -0.8802317976951599, -0.0377717949450016, -0.014049339108169079, 0.09378368407487869, 0.3810037076473236, -0.16462494432926178, 0.0437292754650116, -1.7559094429016113, 0.36225196719169617, 0.5214133262634277, 2.216470956802368, -0.2009316086769104, -0.45349210500717163, -1.289101481437683, -0.21091574430465698, -0.1360652893781662, -0.05067931115627289, -1.5391381978988647, 4.317662715911865, 1.4059091806411743, -1.425876498222351, -0.02296489104628563, 0.08596887439489365, -1.8054075241088867, -1.3245214223861694, 0.22416773438453674, -0.05600005015730858, -0.0584295429289341, 0.17328454554080963, -2.543224811553955, 0.06588750332593918, 0.02783268131315708, 0.7893716096878052, -0.05847390741109848, -0.1454227864742279, 0.8705894351005554, 0.9607207775115967, -0.21866106986999512, 0.3799861967563629, -0.22598984837532043, -0.6197569966316223, -1.6524430513381958, 0.08284209668636322, -0.12274760752916336, 1.1171470880508423, -0.04295014962553978, 0.06200858950614929, 1.1091973781585693, -0.010481583885848522, -0.08635787665843964, 0.4251561164855957, 0.06292898207902908, -0.4580546021461487, 1.033778429031372, -0.02481750212609768, -0.6882240772247314, 0.2771175801753998, -0.13413186371326447, -0.562558650970459, 0.04090070351958275, -0.15044300258159637, 0.3539731204509735, 1.039190411567688, 1.3975496292114258, 0.3554437458515167, -0.7511732578277588, 0.26016369462013245, 3.248589515686035, -0.19778765738010406, -3.9677302837371826, 0.41226890683174133, -0.5494831800460815, 0.2155783772468567, -0.03651009500026703, -0.1451360583305359, 1.0504714250564575, 0.02614106796681881, 0.47837257385253906, -1.0278159379959106, 0.08392538130283356, -0.6221475005149841, 0.3831348121166229, 0.20136137306690216, -0.03910033777356148, -0.627065896987915, -0.16555428504943848, 0.16241183876991272, 0.3772202432155609, -0.0779007151722908, -0.3673595190048218, 1.098307728767395, 1.0029181241989136, 1.2859915494918823, -0.10106184333562851, -0.22859981656074524, 0.016584288328886032, 0.8695463538169861, 0.41031619906425476, -0.8378734588623047, -0.0016405083006247878, -1.1290491819381714, -0.30651137232780457, -2.482680320739746, -0.5814640522003174, -0.25392404198646545, 1.09451162815094, 0.7502029538154602, -0.794691264629364, -0.46126797795295715, -0.09389114379882812, 0.13184353709220886, -1.0546231269836426, -0.02030092477798462, 0.03869165852665901, 0.4741491973400116, -2.956918716430664, 0.42490339279174805, -1.5787280797958374, -0.3978217542171478, 0.06465469300746918, -0.43616801500320435, -0.2757869362831116, 0.00030999109731055796, -1.176466941833496, -0.08085427433252335, 0.5949258804321289, 0.4045051634311676, -0.5567569136619568, -0.4945513606071472, 0.7844570875167847, -0.16121532022953033, 0.32679885625839233, -0.5466631054878235, 0.1507793515920639, 7.588165118477264e-08, 0.06686119735240936, -0.07302411645650864, 0.006863217335194349, -0.49824535846710205, 0.026739399880170822, 2.2013630867004395, -0.957729697227478, 0.19667787849903107, -1.410245418548584, 0.32454749941825867, -1.237914800643921, -0.8281145095825195, -0.14738577604293823, 0.08612558245658875, 0.3503214716911316, 2.8660497665405273, 0.1067558228969574, 0.29057401418685913, 0.9681909680366516, -0.478515625, 0.5750458836555481, -0.2814222574234009, 1.1285747289657593, -0.49537214636802673, 0.004462982062250376, -0.2811116874217987, 0.550179123878479, 1.6539727449417114, 2.102466106414795, -0.021118661388754845, 0.7354507446289062, 0.91267991065979, -1.6999818086624146, 0.11182045191526413, 0.0625261440873146, 0.18099825084209442, -0.012908967211842537, -1.1313927173614502, 0.42021653056144714, -0.01434740424156189, -0.07191786915063858, -0.10013936460018158, 0.4656548500061035, 0.07172553986310959, 0.42002835869789124, -0.2918965220451355, -0.0760374441742897, -0.06800971925258636, -0.2555464208126068, -0.5454566478729248, 0.1511445790529251, -1.8612275123596191, -0.04725325107574463, -0.05635758116841316, 0.0752715989947319, 0.31963858008384705, -0.7691702842712402, -0.392600417137146, 0.025493109598755836, 1.3952383995056152, 1.1646544933319092, 0.5155460238456726, 0.25321123003959656, 1.1952295303344727, 0.15286721289157867, -0.06064093858003616, 0.291803777217865, -0.002920090686529875, 0.008219403214752674, -1.784614086151123, -0.5240612030029297, 0.2814871370792389, -1.5019749402999878, 0.5757526755332947, -0.3986000418663025, 3.4147109985351562, -0.6847640872001648, -0.5438856482505798, -0.008844027295708656, -0.8491054773330688, -0.36805570125579834, -0.1482369303703308, -0.12582938373088837, 0.16675134003162384, -0.037106264382600784, 0.03522329404950142, -0.3724641799926758, 0.029395457357168198, 0.4507804214954376, -0.4388304650783539, 0.6448349356651306, -0.6159912943840027, 0.6104714870452881, -0.21738985180854797, -0.04287604242563248, 0.4977593719959259, -0.04680899530649185, 0.6126223206520081, -0.5003851056098938, 0.03901970013976097, -4.4020676612854, -0.22520120441913605, 0.06462840735912323, -0.18630647659301758, 0.6987434029579163, -0.03790127485990524, 0.40491366386413574, -0.1451413333415985, -0.8231491446495056, -0.9431095123291016, -0.1303214430809021, 0.568559467792511, -2.240875482559204, -0.07573515921831131, 0.09607324749231339, -0.24629229307174683, 0.2608168125152588, -0.6185001134872437, -0.11560140550136566, -0.41123735904693604, 0.22360016405582428, 1.5218162536621094, -0.2680205702781677, 0.04526134580373764, -0.40339934825897217, 0.5442195534706116, -0.10522397607564926, -0.7460780739784241, 1.8052184581756592, -0.705564022064209, 1.07422935962677, -0.06060625612735748, -1.1788893938064575, 0.7810779213905334, 0.505048394203186, 0.0007957690395414829, 0.03619810938835144, 0.22330929338932037, -0.1895161122083664, 0.13067477941513062, -0.9558598399162292, -1.4473230838775635, 1.7934740781784058, 0.4935334324836731, 0.05117213726043701, 0.3060157299041748, -1.230534553527832, -0.7982179522514343, 0.19892406463623047, 0.34512490034103394, 0.044320106506347656, -3.3178365230560303, 0.643936276435852, -0.14355792105197906, 0.2530077397823334, -0.8691992163658142, -1.2403483390808105, 0.2254369705915451, -0.06639611721038818, -0.6711375117301941, -0.4624791443347931, -0.5731698274612427, -1.7761096954345703, -0.7276530265808105, -1.6754896640777588, 0.5457276105880737, -0.874628484249115, -1.2095246315002441, -2.708468437194824, 0.024431737139821053, 0.059464503079652786, -1.2864629030227661, 0.5135637521743774, 0.20816850662231445, -0.9894946217536926, -4.278733730316162, -1.432868480682373, -0.1832333356142044, 0.002371658571064472, 1.057703971862793, 1.0917531251907349, 0.15036515891551971, 0.11599963158369064, 0.0875510722398758, 0.37674590945243835, -0.7124460339546204, 0.0020011207088828087, 0.8013983964920044, -1.815582036972046, -0.08728096634149551, -1.4126005172729492, 0.3436833918094635, -0.07143459469079971, -0.24964399635791779, 0.350727915763855, -2.609405517578125, -0.009189167059957981, -0.04269922897219658, -0.40386074781417847, -0.1003246158361435, -0.9123458862304688, 0.16245567798614502, -0.1800551414489746, -0.21855975687503815, 0.11028845608234406, -0.10175351053476334, 0.004027851857244968, -0.09208495169878006, -0.9627819061279297, 0.2029232233762741, 0.22064408659934998, 0.08346796035766602, -0.13492394983768463, -0.9110859036445618, -3.5740966796875, -4.4280877113342285, -0.20046336948871613, -0.0037850451190024614, -0.36554256081581116, -0.42832767963409424, 0.07197486609220505, -1.5833691358566284, -1.6474241018295288, 1.6170576810836792, -1.8791446685791016, 0.008918462321162224, 3.043895721435547, 0.9656650424003601, 0.13501042127609253, -0.02994280681014061, -0.038412611931562424, -0.6596073508262634, -0.931853175163269, 0.30059266090393066, 2.420558452606201, -0.8596950173377991, 0.038702864199876785, -0.04534878209233284, 0.06230733171105385, -0.06284619867801666, -0.062334559857845306, 0.11087214946746826, 3.0160460472106934, 0.8803826570510864, -0.15667852759361267, 0.23334446549415588, 1.6978379487991333, -4.955868721008301, -0.09335315972566605, -1.4130315780639648, -0.12942858040332794, 2.0305066108703613, 0.42552846670150757, 1.1391923427581787, -0.35106146335601807, 0.7472952604293823, -0.1123797595500946, 0.7084975838661194, 0.052132438868284225, -0.37289154529571533, 0.018406027927994728, -3.835118055343628, 0.016354180872440338, -0.022496933117508888, -0.22914676368236542, -0.4672413766384125, 1.3214032649993896, 0.043439432978630066, -0.002891130978241563, 0.8761483430862427, 2.0436089038848877, 0.011945491656661034, 0.2756982147693634, 1.222555160522461, 0.14046485722064972, -2.641042709350586, 0.9032630324363708, 0.3249564468860626, -0.359773725271225, 0.03867337107658386, -0.6022847294807434, 0.04399214684963226, -0.9522125720977783, -0.3021887242794037, 1.3612297773361206, -1.7236353158950806, -0.08864793181419373, -0.15756209194660187, -2.898101329803467, 0.018001055344939232, -0.11335815489292145, -0.2703801393508911, -1.2193803787231445, 1.508436679840088, 0.7735921740531921, 4.161794185638428, -0.6123855113983154, -3.8604483604431152, 2.8783211708068848, 0.23544682562351227, 0.10235423594713211, 0.4379298686981201, -0.09070824831724167, -0.09795577079057693, 0.1382654458284378, 1.0765466690063477, -0.5143277049064636, 0.3713170289993286, -0.5018571615219116, -1.5288875102996826, -0.30227893590927124, 0.3983879089355469, 0.7462455630302429, 0.3653685450553894, 0.03414119780063629, 2.194397449493408, -0.009467212483286858, -1.0826369524002075, 0.24996915459632874, -0.19804224371910095, 0.036492448300123215, 0.7915881276130676, 0.5705655217170715, -0.9771366119384766, -0.33042675256729126, 0.12943965196609497, 0.19306562840938568, 0.5233882665634155, 4.331522464752197, -0.3006507158279419, 0.1519131362438202, -0.32638028264045715, 0.005140007473528385, 0.412639319896698, 0.36889204382896423, -3.2297685146331787, -0.15841643512248993, 0.571657657623291, -0.15769393742084503, -1.048362135887146, 1.0671236515045166, 0.3113243281841278, 0.3651626408100128, 0.7401956915855408, 0.23685695230960846, -0.4032929241657257, 0.46560388803482056, 0.10364027321338654, 1.237568974494934, 0.17788684368133545, 0.21191710233688354, -0.4741246998310089, -0.2268897294998169, -2.1387131214141846, -1.4870473146438599, -0.6204844117164612, -0.7866353392601013, -0.006085149943828583, -0.3447946310043335, -0.012390078976750374, -5.08745813369751, 0.6460211277008057, 0.5881274342536926, -0.08746712654829025, -0.7499935626983643, -0.7913612127304077, -0.5126087069511414, 1.407486915588379, -1.2331104278564453, 0.29486164450645447, -0.6585029363632202, 0.5838649272918701, 0.9923498034477234, -0.10619974881410599, 0.22652171552181244, -0.32249802350997925, -1.2281620502471924, 0.9761092662811279, -0.5219710469245911, 0.1617480218410492, -2.614572048187256, -0.03960756957530975, -0.7888485789299011, 1.0565139055252075, -0.34520140290260315, 0.23919866979122162, -0.10747655481100082, -0.5754149556159973, -0.26046204566955566, 0.31131991744041443, 0.11969025433063507, -0.3343501687049866, -0.5575893521308899, 0.2509046494960785, -1.690020203590393, 1.44588303565979, -3.95388126373291, -0.17245616018772125, 1.3343586921691895, 0.4327012002468109, 0.6374894976615906, -0.23092952370643616, -0.11681751906871796, -0.41074684262275696, 1.1303426027297974, 0.3786884546279907, 0.32350343465805054, 1.464367389678955, -0.21303904056549072, 0.3299334943294525, 0.8554310202598572, -0.14332303404808044, -0.7024347186088562, -0.051432929933071136, 0.23222912847995758, -2.022653341293335, -0.1513860523700714, 0.08081818372011185, 0.9883526563644409, 1.9351255893707275, -0.29198986291885376, -2.2955338954925537, 0.3129088282585144, 0.2853751480579376, -0.006392996292561293, 0.004192555323243141, -0.9458877444267273, 0.11290376633405685, 2.5543694496154785, -0.6671249270439148, -0.005644524935632944, -0.23784436285495758, 0.06300012767314911, 0.06687608361244202, 0.036445777863264084, -0.6278456449508667, -0.01308653224259615, 1.4156359434127808, -5.5691478337394074e-05, -1.8677045106887817, 1.0286262035369873, 0.6963935494422913, 0.0958985835313797, 0.35549846291542053, -0.7911774516105652, 1.660622000694275, 0.14369447529315948, 0.03496942296624184, -0.6294000148773193, -0.12323647737503052, 0.13479630649089813, -4.068713188171387, 2.9784600734710693, -1.4154828786849976, -0.39028871059417725, 0.012850441969931126, 0.42154690623283386, -0.3901282250881195, 0.04453026130795479, -0.026151170954108238, -0.4902530014514923, -0.42596694827079773, -1.7356297969818115, -0.07533083111047745, 0.2235720455646515, 0.5340836644172668, -0.0028979142662137747, 0.5154749155044556, 0.6586118936538696, -0.1447272151708603, -1.283992052078247, 0.8053575158119202, -0.06233476474881172, 0.21403230726718903, 0.22316548228263855, -0.29085928201675415, 1.4345954656600952, -0.9595453143119812, 0.7392696738243103, -0.5961792469024658, 0.9777206778526306, 0.19476507604122162, 0.23823662102222443, -0.5949112772941589, -0.02431154064834118, -0.41271770000457764, -0.0029763502534478903, 0.016018379479646683, -0.13586480915546417, -0.06228106468915939, 0.5750735998153687, 1.4458909034729004, -0.40094879269599915, 0.326549232006073, 0.25236886739730835, 0.06293567270040512, 0.06923497468233109, -0.0339038260281086, -0.18458214402198792, 0.6746204495429993, 0.5411702990531921, 0.15050816535949707, -0.08729104697704315, 1.3285799026489258, 1.1129099130630493, -0.842621922492981, 0.25267350673675537, 0.6767174005508423, -2.541806697845459, 0.04956226050853729, -0.21146222949028015, -0.2955520451068878, -0.0007578157819807529, -0.02345316670835018, -0.203611820936203, 0.1817951798439026, -1.3785653114318848, -0.0581623837351799, -1.003673791885376, -0.8923207521438599, -0.020382020622491837, 0.055067192763090134, 2.585103988647461, -0.4126076102256775, -0.34206345677375793, 0.4864084720611572, -0.2013092041015625, 0.12838803231716156, -0.39482277631759644, -0.5891886353492737, -0.3082105815410614, -0.025904584676027298, -1.532886266708374, -0.1537434160709381, 0.1371367871761322, 1.1015703678131104, 0.15090662240982056, 0.9569439888000488, -0.42928075790405273, 1.2295821905136108, 0.39764732122421265, -0.6770644187927246, 0.3632795214653015, 0.0633082240819931, 0.9952582716941833, -1.7527376413345337, 0.031117558479309082, -1.225368857383728, -1.3102457523345947, -0.820920467376709, -0.5967962741851807, 0.3558468818664551, 0.014829468913376331, -3.0588929653167725, -0.1913754791021347, -0.5328381657600403, -0.761786699295044, -0.6087256669998169, -0.21153655648231506, -1.1741759777069092, 0.07468798756599426, 2.0703766345977783, -0.022476283833384514, -0.09419647604227066, 0.05095844343304634, 0.7451069951057434, 0.4653880298137665, 0.6025288701057434, 0.08255744725465775, -0.9400084018707275, 1.6440422534942627, 0.13973604142665863, 0.32626622915267944, -0.7203707695007324, -0.011780920438468456, 0.6161587238311768, -0.686415433883667, -0.2234797179698944, -1.241441249847412, -0.22632965445518494, 0.44412797689437866, 0.4020270109176636, -0.13570618629455566, -0.5329153537750244, -0.8167924284934998, -0.6654664278030396, -0.43448057770729065, -0.13164028525352478, 0.13435669243335724, 1.0941640138626099, -0.3119169771671295, 1.4024738073349, 0.631950855255127, 0.30583271384239197, 0.12785741686820984, 0.10405553877353668, 0.2988584339618683, 2.3187997341156006, 0.32927680015563965, -0.23425035178661346, -0.07044162601232529, -0.4812418520450592, 0.004453486297279596, -0.3401297926902771, -0.18710532784461975, -0.6603278517723083, 2.32387113571167, 0.8637633323669434, 0.144260436296463, 0.23779043555259705, -0.4843408763408661, 0.46512341499328613, 0.2497955560684204, 0.321753591299057, 0.1255631297826767, 1.3955341577529907, -0.6104860305786133, 0.3810981810092926, 1.3320633172988892, -1.8754862546920776, 0.23954565823078156, -0.04239330440759659, -0.004929115064442158, 0.0033238574396818876, 2.575421094894409, -3.9407107830047607, 0.09194529801607132, 0.3974655270576477, -0.3404730558395386, 0.5960319638252258, -0.035097312182188034, -0.018207799643278122, -0.23447056114673615, -0.6361117362976074, -0.5221827626228333, -0.11219316720962524, -0.8773871660232544, 0.14790691435337067, -2.6445112228393555, 1.328290581703186, 0.14546020328998566, -0.6614800095558167, 0.35374635457992554, 0.2293066531419754, -0.41214805841445923, -0.703438401222229, 0.004466216545552015, 1.1172878742218018, -1.5727382898330688, 0.17737922072410583, -0.03645610064268112, 0.510322093963623, 0.018405545502901077, 0.07717984169721603, 0.005998728796839714, -0.05751883611083031, -1.230770230293274, 1.1097464561462402, -1.7203342914581299, -0.5695503354072571, -0.0122690973803401, 2.4938085079193115, 1.1959741115570068, -0.8733817934989929, 0.8254749774932861, 0.5819950699806213, 0.0041311499662697315, 0.030163999646902084, 0.7927274107933044, -3.047022581100464, -0.29553771018981934, -0.32133620977401733, -0.39622142910957336, 1.2289937734603882, -0.16481170058250427, -0.06201739236712456, -0.3380448818206787, 0.6151745319366455, 0.48195675015449524, 0.7712190747261047, 0.5578617453575134, 0.8984948992729187, -0.4081811308860779, 0.2744867503643036, -3.1318929195404053, 2.025822162628174, 2.4738118648529053, -0.007297156844288111, 0.46133726835250854, -0.03589951619505882, -0.373433381319046, 0.4162374436855316, -1.034753680229187, -1.019638180732727, 0.5209563970565796, 0.9128844141960144, 0.18463778495788574, 0.09726246446371078, -0.10680891573429108, 0.04815436154603958, 0.055010829120874405, 0.5420128107070923, 0.6086923480033875, 2.628913640975952, -1.0566614866256714, 2.0801048278808594, 1.3121002912521362, 0.8476319313049316, 0.0682360902428627, 0.3076570928096771, 1.3216568231582642, 0.09255585819482803, 0.33329370617866516, -0.14595742523670197, -1.3263201713562012, -0.05446109548211098, -1.6263234615325928, -2.1649553775787354, -0.08119247108697891, -0.9002618193626404, -3.4475314617156982, -0.11852893978357315, 0.5064523816108704, 0.39324304461479187, 0.08893387764692307, 0.34191763401031494, 0.6591222882270813, -0.5931927561759949, 0.8353610038757324, -0.9207342267036438, 0.293269544839859, -0.8720526695251465, 0.42921358346939087, -0.326993852853775, 1.8191419839859009, -1.424971103668213, 2.1206235885620117, 1.223570704460144, 2.19901704788208, -0.09945330768823624, -1.4302291870117188, 0.9255566596984863, -0.23246900737285614, 0.3390448987483978, -0.07200809568166733, 1.0252022743225098, 0.001572320587001741, 2.8538365364074707, 0.03956703469157219, 0.15053297579288483, -1.0636614561080933, 0.024823974817991257, -0.03976532071828842, -0.05818552151322365, 0.8992237448692322, 0.19814683496952057, 2.0487334728240967, 1.2947580814361572, -1.0383106470108032, -0.7155818343162537, 0.10578848421573639, 0.029695207253098488, -1.1394975185394287, -0.541569173336029, -0.040831729769706726, 0.42323678731918335, -0.39593642950057983, -0.23064057528972626, -0.18141917884349823, 0.12506653368473053, -0.7286486625671387, 0.3927314281463623, 0.004929174669086933, 0.09924065321683884, -0.13387736678123474, -0.340421199798584, 0.38265979290008545, 1.5531620979309082, 0.04439239576458931, -0.02576782926917076, 0.22458645701408386, 0.03594692796468735, -0.1206108033657074, 0.7720531225204468, -0.9945895075798035, 2.138579845428467, -0.026536719873547554, 0.49140465259552, 0.4466625154018402, -0.2887074947357178, 0.02860025130212307, -2.371142864227295, 0.7040372490882874, -0.11004624515771866, 0.8111101388931274, -0.0500619076192379, 0.10159890353679657, -0.08428044617176056, -0.004075829405337572, 0.4293465316295624, -0.3525276780128479, 0.5667599439620972, 0.043595656752586365, 0.2446414679288864, -0.0013113946188241243, -0.5658065676689148, -1.141288161277771, 1.1434972286224365, 0.14418405294418335, -0.04122287780046463, -1.8322306871414185, -0.10091749578714371, 0.1078934594988823, 0.01071557030081749, 0.4761335253715515, -0.97250896692276, -0.2389003485441208, 0.6238968372344971, -0.566029965877533, 1.3146255016326904, -2.5374603271484375, -0.3850404620170593, -0.7874044179916382, -2.3239963054656982, 0.3932482898235321, -0.017910940572619438, 0.19704510271549225, 0.6347514390945435, -1.2221587896347046, -1.822770357131958, -0.8486023545265198, 1.5444666147232056, -1.9469127655029297, 0.6478484272956848, 0.18456070125102997, -0.2637518048286438, 0.49823886156082153, -0.8076795935630798, 0.14761784672737122, 0.731623113155365, 0.20606189966201782, 0.4117402732372284, 0.3284560441970825, 0.26334384083747864, -1.2203161716461182, -0.6835203766822815, -0.533007800579071, -0.3150845170021057, -0.6198151707649231, -0.6890398859977722, -0.9992149472236633, -1.4720640182495117, 0.12644962966442108, -0.3561735451221466, 0.010616494342684746, -3.1362218856811523, -0.46797579526901245, -1.8743798732757568, 0.043830472975969315, 0.5220490097999573, -0.21112525463104248, -1.0895304679870605, 0.6256885528564453, 0.008676815778017044, -0.010196922346949577, -1.275960922241211, -2.095353603363037, 0.311927855014801, -1.0090713500976562, 0.8533506989479065, -0.0068162959069013596, 0.8662872314453125, -0.04791269078850746, 0.44808611273765564, -0.19691148400306702, -0.10501803457736969, 0.4842236638069153, 0.33782467246055603, 1.078117847442627, -0.3124763071537018, 1.1334242820739746, -0.38635385036468506, -0.037776291370391846, 0.13511618971824646, 0.3609386682510376, -0.0812428742647171, -0.273641437292099, 0.09116893261671066, -0.1764196753501892, 0.36983075737953186, -0.030848363414406776, 0.1171955093741417, -0.24156053364276886, 1.2867660522460938, 1.0120785236358643, -0.5673820972442627, -0.3219965994358063, 0.6833032369613647, 0.0007114644977264106, 0.032261986285448074, 0.015134653076529503, 0.32390159368515015, 0.029087359085679054, -0.05223366990685463, -0.48251214623451233, 0.49720147252082825, 0.24042344093322754, -0.9933057427406311, 1.09183669090271, -0.11493576318025589, 0.2935122847557068, 1.0551124811172485, -0.540641188621521, 0.2150719165802002, 0.014651318080723286, 0.520250141620636, -0.12383865565061569, 0.07264410704374313, 0.25135958194732666, -0.10310381650924683, 0.11850309371948242, 0.10818905383348465, 0.5217885375022888, -0.5363336801528931, 0.2491636425256729, -0.004999945871531963, -0.6918996572494507, -0.46480485796928406, -1.5749915838241577, 0.1669350266456604, -0.004189034458249807, -0.7467576861381531, 0.15657712519168854, 0.04880800470709801, -1.3385818004608154, -1.3193690776824951, 0.5024234652519226, 0.34045255184173584, -0.005285840015858412, -0.6131711006164551, -0.20480865240097046, -0.061719734221696854, 0.02687964029610157, -0.1360604166984558, 0.02882879599928856, 0.3842887878417969, -0.21917183697223663, -2.6239795684814453, 0.0034613802563399076, 0.4145481288433075, 3.9101130962371826, 0.46158742904663086, -0.01730380207300186, 1.99306321144104, 2.442067861557007, 0.5686962604522705, -0.07033520191907883, -0.4256891906261444, 0.089470274746418, -0.393947958946228, -0.2545773983001709, -0.07992171496152878, -1.2518811225891113, -0.06754393875598907, -0.006181533448398113, -0.00011926122533623129, 0.3821120262145996, 1.4859740734100342, 0.18445834517478943, 0.005851754918694496, -0.7732232213020325, -0.07647769898176193, -0.540693461894989, 0.04083478823304176, -0.06293586641550064, 0.6225494742393494, 0.16116677224636078, 0.447060763835907, 0.6492616534233093, 0.012683810666203499, -3.6389758586883545, 0.7850217819213867, -0.006012957077473402, 0.33025887608528137, -0.755700409412384, -1.0892279148101807, -0.1870543211698532, -0.06650577485561371, 0.2775878608226776, 0.1336890608072281, 0.6321585178375244, -1.1849925518035889, 0.03553566336631775, -3.653954029083252, 0.377531498670578, 0.15065352618694305, 1.3702843189239502, 0.07384917140007019, 0.275203138589859, -2.007490634918213, 0.745525062084198, 0.2594706118106842, 0.3209511339664459, -0.10150613635778427, -0.22439616918563843, -0.06389216333627701, 1.742235779762268, -5.551680088043213, -0.41428905725479126, 0.027918562293052673, 0.2316819727420807, -0.0960104689002037, 1.2701207399368286, -2.740355968475342, 0.8327059745788574, 0.07221883535385132, -0.49845731258392334, -1.3950037956237793, 0.6227261424064636, 0.0006512990221381187, 0.4363381564617157, -0.42914459109306335, 0.009088829159736633, -0.4074762463569641, -0.14903710782527924, 2.3584372997283936, 0.9027033448219299, -0.6933293342590332, 0.18021321296691895, -0.03599333390593529, 0.3052918314933777, 0.5861602425575256, 2.034838914871216, 0.8359817266464233, 1.288769006729126, -2.897656202316284, 0.8490989208221436, -5.2703752517700195, -0.021915512159466743, 0.3702660799026489, 0.3317015767097473, -0.19290418922901154, -0.007063071243464947, -0.6954067945480347, -0.1868211030960083, -0.09794320166110992, -1.9633705615997314, 0.5402354001998901, -0.22185571491718292, -0.3073960840702057, 0.9768303632736206, -0.019468748942017555, 1.5615496635437012, 0.02352946251630783, 1.2175445556640625, 0.08648928254842758, 2.318892002105713, 0.5774506330490112, 0.23453976213932037, -1.8037906885147095, 0.6852105855941772, -1.1612920761108398, -0.31267955899238586, 0.5600996613502502, 0.45413288474082947, -0.8053516745567322, 0.38259178400039673, -0.3104136288166046, -0.044519416987895966, -0.6779807209968567, 2.208085298538208, -2.4435513019561768, 0.20744270086288452, -0.4401482939720154, 0.12143179774284363, -1.528513789176941, -0.40176472067832947, -1.9187697172164917, -0.02568064257502556, -0.3788875937461853, 0.011327764019370079, -0.00805453211069107, -0.18424740433692932, -0.03805781528353691, 1.0908936262130737, 0.58223557472229, -0.05036553740501404, 1.3081008195877075, -0.2571985125541687, 0.221290722489357, -1.1355831623077393, 0.27926918864250183, -0.21827811002731323, 0.4744691550731659, -0.3663211464881897, 0.13733290135860443, 0.2050936073064804, -0.0675896555185318, -0.10774358361959457, -0.0014908357989042997, -1.0322922468185425, -0.12204320728778839, -1.2406471967697144, -1.3325848579406738, 0.8404481410980225, -1.5074353218078613, -0.9346446990966797, 0.26786696910858154, -4.182183742523193, 0.27791962027549744, -0.7866649031639099, -0.051434487104415894, -0.009423229843378067, 0.2729949355125427, -0.02496054768562317, 0.6292666792869568, -2.068685531616211, 0.4862561523914337, -0.05869382619857788, -0.06613130122423172, -0.6014207005500793, 1.388913869857788, -0.05499432235956192, 0.8818926215171814, -0.07097421586513519, 0.9695278406143188, 2.260582208633423, 0.5605109333992004, -0.08348508924245834, -0.0036065136082470417, -0.12990523874759674, -0.003172369906678796, 1.3035520315170288, -0.20902766287326813, -0.7618643641471863, 0.9673054814338684, -0.16705365478992462, 1.1115601062774658, 1.325567364692688, 1.0922168493270874, 0.3690029978752136, -0.7203649282455444, 0.29096049070358276, -0.423728346824646, 0.2929270565509796, -0.011801845394074917, 1.8786053657531738, 0.19270430505275726, -0.009053291752934456, -0.004210012964904308, -0.027777966111898422, -0.0559903159737587, -0.16162744164466858, -0.07032255083322525, -0.0019096395699307323, 0.5128878355026245, -1.1290040016174316, -0.29301631450653076, 0.28997617959976196, -1.266404628753662, 0.041597880423069, 0.29469481110572815, 1.2739640474319458, -0.36789366602897644, 0.29949524998664856, 0.47971826791763306, 0.2457813173532486, 0.8181582689285278, -0.04005570709705353, 0.4429970979690552, 0.27380865812301636, -0.6054901480674744, 0.1439618617296219, -0.11438290774822235, 0.14662551879882812, 0.009450295008718967, 1.3159153461456299, -0.6856899261474609, -0.3457479476928711, 0.5843257904052734, 1.0661836862564087, -0.8302392363548279, 0.025353219360113144, -0.17040590941905975, 0.14462493360042572, 0.6515374779701233, -4.728742599487305, -0.1347002238035202, -0.0004674926749430597, 0.5099493265151978, -4.65875768661499, 0.8764616847038269, -0.7101635336875916, -1.3403363227844238, -0.22526895999908447, -1.2881461381912231, -0.030400531366467476, 0.6289121508598328, 0.1565464287996292, -1.899245023727417, -0.3444333076477051, 0.06500193476676941, 0.09939488768577576, 1.0449624061584473, 1.2379966974258423, -0.6590687036514282, -0.03593304008245468, 1.6502797603607178, -0.15309610962867737, -0.7893863916397095, -2.539720058441162, 0.4225611388683319, 0.6786879301071167, -0.73053377866745, -2.615769386291504, -0.11874530464410782, -0.9132979512214661, 0.03077569231390953, -0.048698049038648605, -0.28996965289115906, -0.4988086223602295, 0.40448620915412903, -1.5554609298706055, 1.3634263277053833, -0.19912801682949066, -1.7533329725265503, 0.6172922253608704, -1.05461847782135, 0.5742765665054321, 1.04301917552948, 0.08030763268470764, -0.12829065322875977, -0.8434620499610901, 0.03265746310353279, 0.04886488988995552, -0.024773279204964638, -0.7945424914360046, 0.053086865693330765, 0.01643175259232521, -0.09989583492279053, 0.8112490177154541, -0.017690682783722878, -0.40464428067207336, 0.9406399130821228, 0.16030825674533844, -2.349675178527832, 0.6907058954238892, -0.0009099555318243802, 0.0315278060734272, -0.5255964994430542, -0.34372571110725403, 0.015079513192176819, -0.42641323804855347, 1.3863513469696045, 0.29719817638397217, -0.022045152261853218, 2.066525459289551, -0.5636649131774902, 1.287284016609192, 0.20894615352153778, 0.09728189557790756, -0.23402492702007294, -0.037527743726968765, 0.45499569177627563, -0.5771936178207397, -0.4063356816768646, -0.38732099533081055, -0.35785701870918274, 0.300496369600296, -2.121572256088257, -1.9760172367095947, -1.7845312356948853, 0.4243101179599762, -2.0246644020080566, -2.3803904056549072, 0.2772271931171417, -0.38952764868736267, 1.448357343673706, 0.05714409053325653, 0.03231440484523773, 0.2363753616809845, -0.10393041372299194, -0.08039377629756927, -0.5010759830474854, 0.1007445901632309, 0.1729244887828827, -0.061513837426900864, 0.2497437745332718, 1.0367953777313232, 0.3142029047012329, -0.2328578531742096, -2.076782703399658, 0.3836267292499542, 0.05533917620778084, 0.2939356863498688, -0.09870337694883347, -0.09463184326887131, 1.0588334798812866, -0.019590601325035095, -0.0027157485019415617, 0.04241379350423813, -0.044969625771045685, -0.11298279464244843, -1.0336791276931763, 0.029153097420930862, 0.6607775688171387, -0.013786385767161846, -0.14223535358905792, 2.175257921218872, 0.006504991557449102, -0.7296701073646545, -2.2481000423431396, -0.03675998002290726, -0.026099853217601776, 0.3227951228618622, 0.0483877919614315, 0.2972787022590637, 0.722622275352478, 0.08244986832141876, -0.6739405393600464, 0.478756844997406, -0.03484850004315376, 1.3588584661483765, 0.7465723752975464, -1.5583744049072266, 0.001622718176804483, 0.10881049185991287, 0.5322746634483337, 0.09263542294502258, 0.172176331281662, -0.993037223815918, -0.6022042632102966, -0.4735978841781616, 1.4666824340820312, -0.7731334567070007, -0.07444246858358383, -0.5240886807441711, -0.08658291399478912, -0.6794613599777222, 0.8612889051437378, 0.018261291086673737, -0.47249317169189453, 0.9941800236701965, 0.37922021746635437, 0.18027575314044952, -0.5846236348152161, -2.7690622806549072, 0.27960941195487976, 0.09873481094837189, 0.5077205896377563, 0.45712733268737793, 0.002467959187924862, 0.2011740505695343, -0.613597571849823, 0.2632782757282257, 0.7227945923805237, 0.07594012469053268, -3.7514212131500244, -0.05141070485115051, -2.152268409729004, 0.3570824861526489, 2.1523289680480957, -1.2130320072174072, 2.0324759483337402, -0.9352374076843262, -0.5041201710700989, 1.5646703243255615, -1.545067548751831, 0.40250059962272644, 0.8626806139945984, -0.026953978464007378, -0.05987377092242241, -0.047031160444021225, 0.6543562412261963, 0.3194424510002136, 1.8629319667816162, -0.040172070264816284, -0.024703387171030045, -1.2654117345809937, 0.09459441900253296, -0.7459196448326111, -0.6552748084068298, 0.9895862340927124, -2.1229515075683594, -0.8296809196472168, 1.3802529573440552, 0.5092523097991943, 0.07194211333990097, 0.11621849238872528, -0.07819286733865738, -3.6501760482788086, -0.7067350149154663, -2.623274326324463, -2.9235875606536865, -0.5449686050415039, 0.3629356920719147, -0.34712010622024536, 3.135236978530884, 1.566730260848999, -0.3658440113067627, 0.8682094216346741, 3.9248828887939453, -0.835091233253479, -0.12538333237171173, -0.7284961938858032, 0.3668769299983978, 0.16170261800289154, 1.1337599754333496, -0.08205264806747437, 0.011182350106537342, 2.029571771621704, -0.46230486035346985, 0.08783439546823502, -0.21023836731910706, 1.038367748260498, -0.31893429160118103, -0.4361414611339569, -0.5839716196060181, 0.20879533886909485, -0.09949624538421631, -0.018944839015603065, 0.8707544207572937, 1.0583438873291016, -1.5361841917037964, -1.1667332649230957, 1.7674894332885742, 0.0011870331363752484, -0.1843678057193756, 1.7149996757507324, 0.11590763926506042, 0.09842833876609802, 0.04551807790994644, 0.16749651730060577, -2.15362811088562, 0.1723363697528839, -0.09570790082216263, -1.0257567167282104, -1.492660641670227, 0.3221766948699951, 0.042335394769907, -0.8794946670532227, -0.04595070704817772, 0.052294205874204636, -1.5244444608688354, 0.15067309141159058, -0.020383266732096672, -0.7643254399299622, 0.24347467720508575, -0.0533529594540596, 0.020021498203277588, -0.8996970653533936, -1.119353175163269, 0.814945638179779, -1.0136889219284058, 0.571489155292511, -0.2527354657649994, -0.008857260458171368, -0.2513793110847473, -1.1347119808197021, -4.590593338012695, -0.44113004207611084, 1.2117661237716675, 0.6584606170654297, 0.15798954665660858, 0.09951169043779373, -0.23845160007476807, 2.260869026184082, 2.047529697418213, -0.2983063757419586, 0.02217312529683113, -0.1388363391160965, -0.4041631817817688, -0.014105057343840599, 0.17331746220588684, 0.032995447516441345, 0.03340340405702591, 0.931760311126709, 0.793128252029419, -0.10721613466739655, 0.23344215750694275, 0.6053382158279419, 1.4356796741485596, -0.23478350043296814, -0.02033325657248497, -0.5483754277229309, 0.23166535794734955, 0.14567768573760986, -2.51285457611084, 0.32266736030578613, -0.029082613065838814, -0.36703169345855713, 1.7422878742218018, 0.7510764598846436, 0.05166470631957054, 1.3646066188812256, 1.2741670608520508, 2.3390891551971436, 0.3097616136074066, -0.16819410026073456, 0.2569403648376465, -3.1619744300842285, 0.9004793167114258, 4.449016571044922, -0.4597110152244568, 0.7226296663284302, -0.2868751585483551, 0.6604620814323425, 0.24445396661758423, 0.041308946907520294, 0.020721718668937683, -1.0709006786346436, 3.372014284133911, -0.07218277454376221, 0.0987575426697731, 0.4700983166694641, -0.041613876819610596, -0.4149167537689209, -0.03753818944096565, -1.297487497329712, 0.11934150010347366, -0.3423249125480652, -2.1968600749969482, -0.06682111322879791, 0.11269543319940567, 0.04672163352370262, 0.26176005601882935, 0.17877548933029175, 0.34850266575813293, 0.01044471561908722, -0.00562813738361001, 0.4957932233810425, 0.274729460477829, -0.02657957375049591, 0.9935102462768555, 1.0451215505599976, 0.06334193795919418, -0.9458637833595276, -0.145472452044487, 0.3597247004508972, 0.05890665203332901, 0.9506869316101074, -1.4386247396469116, -0.5812994837760925, 0.9023874998092651, -0.14415200054645538, -0.05161319673061371, 0.3160350024700165, 0.0038709964137524366, -0.01090445090085268, 1.1028014421463013, -0.33212676644325256, -0.22674833238124847, -0.9729184508323669, -0.7614789605140686, 0.4100223779678345, -0.04519376531243324, 0.3124499022960663, 0.44201406836509705, 0.4432455003261566, 1.069595217704773, -2.578094244003296, 0.6722333431243896, 0.17816844582557678, 0.2921570837497711, 0.13564172387123108, 1.052164912223816, 0.06081725284457207, -0.6369160413742065, 0.8677674531936646, -0.7904117107391357, 0.963388979434967, -0.1391744464635849, -0.2523347735404968, -0.4665306508541107, -1.1361750364303589, 0.7276526093482971, -0.017773602157831192, -0.03199708089232445, 0.2299068421125412, 0.12335406988859177, -0.41876497864723206, 0.06849700957536697, -0.9007359743118286, -0.27760404348373413, -0.7393505573272705, -0.45752060413360596, -0.5303320288658142, -0.016448095440864563, 0.08629923313856125, -1.08534574508667, -0.5469047427177429, -0.3296356797218323, -0.1826523244380951, 1.7064142227172852, 0.48048147559165955, -0.24393399059772491, 0.1500789225101471, 0.16971270740032196, 2.6343555450439453, -0.8767044544219971, 0.07434272766113281, -0.2456626296043396, -0.33251580595970154, 0.13161034882068634, 1.5202984809875488, -0.3755049407482147, 0.5936232209205627, -0.03812167048454285, 0.1770641803741455, -0.009998144581913948, -0.14598409831523895, -1.0505473613739014, -0.056740280240774155, 0.36795639991760254, 0.770599901676178, -0.020670205354690552, 0.2368759661912918, -0.18781918287277222, 1.1849391460418701, 0.38127681612968445, 0.43862882256507874, 0.13725003600120544, -0.02063317969441414, 0.7737572193145752, -2.473992109298706, 0.024985672906041145, -0.5331675410270691, -0.09714952856302261, 3.804609537124634, 1.0084890127182007, -1.0952532291412354, -0.07686565071344376, -0.3056005537509918, -0.3429645597934723, -0.3668144941329956, -1.0293370485305786, 0.01928921602666378, -0.18542331457138062, 0.21984414756298065, 0.5264959931373596, -0.023275135084986687, 0.09413334727287292, -0.030690809711813927, 0.3958459198474884, 1.6728107929229736, -0.49706384539604187, 0.17617136240005493, -0.42298105359077454, -0.20226162672042847, 0.21432971954345703, 0.12790244817733765, -1.3142863512039185, 1.4343345165252686, -0.3501148819923401, 3.006258010864258, -0.530339777469635, -0.17843720316886902, 1.2936533689498901, 0.2675691843032837, 0.0691743791103363, -0.10333005338907242, -1.5412533283233643, -0.0644221305847168, -0.2548302412033081, 1.0797992944717407, -0.6205760836601257, -0.3975762128829956, 1.5338366031646729, -0.09737680107355118, -0.8759097456932068, -0.08495527505874634, 0.22501172125339508, 0.9775574207305908, -0.2858789265155792, 0.0977250188589096, 0.8801408410072327, 0.13093866407871246, -0.02388566918671131, 1.0256389379501343, -1.803228735923767, -0.47456949949264526, -0.00716021191328764, -0.05327948182821274, 0.6658706068992615, -0.3905445635318756, -0.40327373147010803, -0.8533564805984497, -0.2779763340950012, 0.03538995608687401, 0.5726324915885925, 1.1327757835388184, -0.845460057258606, -0.3222387433052063, 0.5087743997573853, 0.7646827697753906, 0.2990477681159973, 0.028478674590587616, 0.25775352120399475, -0.03718778118491173, -0.398306667804718, 0.012178092263638973, 0.2531980872154236, -1.1219120025634766, 1.9866909980773926, 0.003036437788978219, 1.4762017726898193, 0.20913836359977722, -0.04870443791151047, 0.47152578830718994, -0.2527867555618286, -1.3059558868408203, -0.03340675309300423, -0.8724979162216187, -0.033321477472782135], [-0.46199238300323486, 1.551407814025879, 0.010111761279404163, -0.6888368129730225, 1.299340844154358, 2.4356017112731934, 0.6709781885147095, 0.2699124813079834, -0.018520405516028404, -1.203991413116455, 0.4904552102088928, 1.5499253273010254, 0.1966651827096939, 1.2552227973937988, -1.0540130138397217, 0.6391094326972961, 1.906190037727356, 0.7254294753074646, 1.1651973724365234, 0.06293968111276627, -0.25113749504089355, 0.06816748529672623, 2.6488606929779053, 3.9913618564605713, 1.7386425733566284, 0.11354885250329971, -0.8865820169448853, -0.4049201011657715, -0.6787888407707214, 0.007038724608719349, -0.16772666573524475, -0.5241513848304749, 0.6679124236106873, 0.022301843389868736, -0.0920296236872673, -1.0363649129867554, 0.08130054175853729, -0.7352907657623291, 0.010220992378890514, 0.4625040292739868, -0.3469181954860687, 0.08736629039049149, 0.31094563007354736, 0.04462581127882004, -0.09998087584972382, 1.2003593444824219, -0.41142934560775757, -0.205426886677742, 0.15472553670406342, -0.2340460866689682, 2.0806477069854736, -0.8310614228248596, -1.143552303314209, 0.9956544041633606, 0.06631936877965927, -1.2145812511444092, -1.068204402923584, -0.8859497904777527, -0.02388656511902809, -0.206465482711792, -0.30289074778556824, 0.22055605053901672, -1.0639816522598267, 1.3547773361206055, 0.022289808839559555, 0.1649627834558487, 0.26447537541389465, -1.0212041139602661, 0.25534141063690186, 0.3311809301376343, -0.5377776622772217, -2.764439821243286, -0.799318790435791, 0.3764593005180359, -1.1240118741989136, 0.3584790527820587, -0.5274502635002136, 0.1280185729265213, -0.07142510265111923, -0.008858691900968552, 2.1545276641845703, -0.12959501147270203, -2.2149477005004883, 0.7232380509376526, -3.064746379852295, -0.019906923174858093, 0.06621138006448746, -0.9122136831283569, -1.941900610923767, -0.08231709897518158, 0.016550643369555473, 0.8732299208641052, 0.01068808976560831, 1.1463391780853271, 0.009023883379995823, 0.5661420822143555, -0.1275988668203354, 0.35367295145988464, -1.3003499507904053, -0.010702296160161495, 0.7135069966316223, -5.909862518310547, -0.992195188999176, -0.09783022105693817, 0.08917342871427536, -0.058096546679735184, 0.894602358341217, -0.0812097117304802, 0.003650367958471179, -0.17916743457317352, 0.1621684432029724, 1.3257436752319336, 0.21323931217193604, 0.00461475970223546, 0.7250922918319702, -2.5381722450256348, -0.2741607427597046, -0.1544259637594223, 0.13951337337493896, 0.1820458471775055, -0.6999840140342712, -0.17858029901981354, 0.03581644594669342, -0.4304654896259308, 2.068845272064209, 0.11015065014362335, 0.9372157454490662, 0.34679800271987915, -0.13935275375843048, -0.270106703042984, 0.30667930841445923, -0.09315523505210876, 0.4696528911590576, 0.5610957145690918, 0.2826499342918396, 0.15567955374717712, 0.013352541252970695, 0.06839055567979813, 0.03473202511668205, -0.08671703189611435, 0.14963480830192566, -0.08860015124082565, -0.2690981924533844, 1.761563777923584, -0.018174419179558754, -0.38657617568969727, 0.08560794591903687, 0.3733116090297699, 0.15081705152988434, 1.6474361419677734, -5.533938884735107, -0.933515191078186, -0.4002063274383545, 0.257019966840744, 1.2826974391937256, -0.9484924077987671, 0.17946098744869232, -0.9602534770965576, -0.40680423378944397, 2.6037356853485107, 2.307013988494873, -0.5055482983589172, -0.06557121872901917, 0.479358434677124, -0.05390370637178421, 0.23443682491779327, 0.2346964180469513, -0.012161419726908207, -1.770149827003479, -0.45624563097953796, -0.8284744024276733, 0.3047039806842804, -0.10248072445392609, -0.6291127800941467, -4.5976033210754395, 0.12908804416656494, -0.36924001574516296, -0.13723553717136383, 0.1854722797870636, 1.4531928300857544, -0.2708183228969574, -2.5237114429473877, -3.1584317684173584, 0.9513974785804749, 1.3743398189544678, -0.24401454627513885, -0.000825647555757314, 0.008151024580001831, 1.475016713142395, 0.00508991489186883, 1.3099037408828735, 2.417459011077881, 0.7425826787948608, 1.8442556858062744, -0.021882055327296257, 0.04293394088745117, -0.36440032720565796, -0.010963333770632744, 0.039771854877471924, 1.4602564573287964, 0.06437802314758301, 0.04072406515479088, 0.04773302003741264, 0.19948071241378784, -1.422360897064209, 0.0678660199046135, 1.6383686065673828, -0.34199458360671997, -0.47167539596557617, 1.509752631187439, -0.1758713573217392, -0.12067312002182007, -0.6362755298614502, -0.1490475982427597, 0.04975792020559311, -0.16264134645462036, 2.117171049118042, -0.9076764583587646, -0.8449662923812866, -0.5398913025856018, 0.2166319489479065, 0.6863411664962769, -0.2541064918041229, 0.059153780341148376, 1.1931756734848022, 0.06813745945692062, 0.8199276924133301, -0.6120481491088867, 1.3167319297790527, 1.2995014190673828, -0.01956358551979065, -0.44690850377082825, -0.07771655172109604, -0.9360527396202087, 0.2909988760948181, 0.8427644371986389, 0.09122752398252487, 0.3794732689857483, 0.2652181386947632, 0.12554135918617249, -1.7092307806015015, 0.051437363028526306, -0.056813303381204605, 0.14486590027809143, -0.027104679495096207, -0.1666562557220459, -0.05958511680364609, 0.02750430628657341, 0.013243049383163452, -1.1371153593063354, -0.5385921001434326, -1.0053104162216187, 0.672966480255127, 0.04762544855475426, 0.4284876585006714, -0.07547149062156677, -0.1392517387866974, -0.18170039355754852, -0.17456617951393127, -0.06605387479066849, -0.20561130344867706, -1.5063284635543823, -0.6234776973724365, 0.48789873719215393, 1.0530381202697754, 0.7087141275405884, -1.545822262763977, -0.0912487804889679, 2.53456711769104, 0.16958941519260406, 0.4181894361972809, 0.16506879031658173, 0.32234352827072144, 0.06126510351896286, -0.9187059998512268, 0.0012102312175557017, 0.31161898374557495, 0.2761380672454834, -0.009096501395106316, -0.6336629986763, -0.572044849395752, 0.23136895895004272, -0.39052614569664, -1.1379101276397705, 0.2780020534992218, 0.5813612937927246, -0.8981058597564697, 0.034040842205286026, -0.08765001595020294, 1.163352131843567, 1.7970318794250488, -0.3155655860900879, -0.012045217677950859, -0.6554547548294067, -0.03672260418534279, -1.3827368021011353, -0.7569499611854553, -0.5815678238868713, -1.239609718322754, -0.4251185953617096, 0.5375381708145142, 1.2356890439987183, 0.44930827617645264, -0.28261813521385193, 0.13525214791297913, -0.8104739189147949, -0.7030368447303772, 1.7570384740829468, 3.7849514484405518, 0.05618276819586754, -0.1488296538591385, -1.9762688875198364, -0.0010933532612398267, -1.1599128246307373, -1.4904072284698486, 1.423951268196106, -0.3929559290409088, -0.5409361124038696, -0.17267371714115143, -1.9450232982635498, -0.8795672059059143, 0.2124834954738617, -0.02114204689860344, 0.6286926865577698, 1.0052762031555176, 0.42140135169029236, 0.5574919581413269, 0.36090442538261414, 1.3192014694213867, 1.5370173454284668, -0.2069660723209381, -0.06604059040546417, -0.060432907193899155, -7.422836006298894e-06, -0.8917525410652161, -0.30789557099342346, -0.020262833684682846, -1.375683307647705, -0.0472087599337101, -0.09987393021583557, -0.32553598284721375, -1.1583359241485596, -1.1192947626113892, 0.682514488697052, 0.10263193398714066, -0.16518105566501617, 0.3036719858646393, -0.24521972239017487, 0.06018400564789772, 1.252920389175415, 0.41841793060302734, -0.34688127040863037, 0.21587008237838745, 0.6175238490104675, -0.5099955797195435, 0.2423693984746933, -0.42589500546455383, -0.7085950970649719, -0.024705342948436737, -0.20612916350364685, 0.2460600584745407, 3.1321353912353516, 1.2900382280349731, 0.00999817531555891, 0.6601978540420532, 0.2478361576795578, 0.7047023177146912, -0.09027491509914398, 0.4987749457359314, 1.8862078189849854, 0.28457361459732056, -0.5619986653327942, -1.0692744255065918, -0.04056377708911896, 0.676474392414093, 1.1453946828842163, 0.13709747791290283, -0.03172725439071655, -1.5165588855743408, -0.7270094156265259, -0.08947072923183441, -0.0859389677643776, -0.13088230788707733, -0.2944023311138153, -1.1263842582702637, -0.4105096459388733, 0.08656696230173111, -0.022685304284095764, 0.775928795337677, 0.41438502073287964, -0.9718917012214661, -0.6048624515533447, -0.2132299393415451, -2.544916868209839, 1.5735234022140503, -0.824665904045105, -0.5040682554244995, 0.18311099708080292, 0.16403287649154663, 2.3875303268432617, -1.0412276983261108, 0.002148644533008337, 0.22196374833583832, 0.0755256935954094, -2.9289188385009766, 1.0886348485946655, -1.0201257467269897, -0.8407723903656006, 0.16520871222019196, 1.382118821144104, -1.3467819690704346, -0.7042582035064697, -0.012048349715769291, -0.7763453125953674, -0.3470766246318817, -0.6007497310638428, 0.06517774611711502, -0.008623036555945873, 0.0759645402431488, 0.8631992340087891, 0.2880846858024597, -0.16376633942127228, -0.5415377020835876, -0.6627540588378906, -0.3269796669483185, 0.22253473103046417, -0.6984485983848572, 0.5825937390327454, 0.16454358398914337, -0.9508443474769592, -0.6242950558662415, 0.5831446647644043, 0.6366829872131348, 0.06331700086593628, -2.1359145641326904, -0.2711425721645355, 0.6091197729110718, -0.6538780927658081, -0.2778513729572296, -0.9495990872383118, -0.14951130747795105, 0.046728719025850296, -0.3246459662914276, 0.3080999553203583, -0.4991266429424286, -2.1456573009490967, 0.40215766429901123, 0.15163281559944153, 0.0887640193104744, -0.06441497802734375, 0.474783331155777, 0.33845293521881104, -0.06943576782941818, 1.1619460582733154, 0.04345228895545006, 3.396951913833618, 0.5024036765098572, 1.7835413217544556, -0.35251227021217346, -0.36384204030036926, 0.4051995873451233, -1.178765058517456, 0.07410771399736404, 0.1012757197022438, -0.7592025995254517, -0.26477429270744324, -1.4026250839233398, 0.5467703342437744, 0.8741171360015869, 0.001100052148103714, -0.09437008202075958, 1.3672033548355103, 0.21217778325080872, -0.21374423801898956, 1.9257268905639648, 0.5462110638618469, 0.3486323952674866, -0.7230477333068848, -0.15551531314849854, 0.08325858414173126, -0.17659403383731842, 2.289935350418091, 0.09564556181430817, 0.04268854111433029, -0.053020283579826355, -0.9756733179092407, -0.35009655356407166, 0.0008233940461650491, 0.06617415696382523, 0.9417895078659058, -0.9330214262008667, -0.33642280101776123, -0.20039336383342743, 1.096733808517456, -1.952242374420166, -2.185497283935547, -1.3334126472473145, -0.3049916923046112, -0.5116592049598694, -1.945453405380249, 0.2783365845680237, 1.3001104593276978, 0.7877201437950134, 0.10058959573507309, -0.08178169280290604, -2.4970157146453857, 0.4072747528553009, -0.042705144733190536, -0.6341438889503479, 0.7492188215255737, 0.6704248189926147, 0.030242741107940674, 0.002809504047036171, 0.5522543787956238, -0.7373965978622437, 0.004019284155219793, 0.1404932737350464, -0.8027005195617676, 0.587417483329773, -0.6252688765525818, 0.007188440766185522, -0.2036203294992447, -1.5244065523147583, 2.2880492210388184, -1.8640214204788208, 0.3198184370994568, 0.0010688666952773929, -0.19378764927387238, -0.25420087575912476, 1.9195036888122559, 0.01543401088565588, -0.49775704741477966, 0.7570745348930359, 0.1031246930360794, -0.8631870746612549, 0.1468905210494995, -0.07828779518604279, 0.9977545142173767, 0.391789972782135, 0.36605304479599, 0.0030828742310404778, -0.02829885110259056, 1.7609870433807373, -0.5742896199226379, 0.3040241003036499, 0.05523674190044403, 0.032987792044878006, 0.5733916163444519, -0.6823549866676331, 0.47581785917282104, 0.028065772727131844, 0.09087584167718887, -0.04421639442443848, -0.9645606875419617, 0.31395986676216125, 1.874843955039978, -0.13919222354888916, -1.755767822265625, 2.4911510944366455, -0.4405190348625183, 0.8751518130302429, 3.156848192214966, 0.22005781531333923, -0.3581902086734772, -0.01504931878298521, 0.1874975860118866, 0.11227299273014069, 0.14552748203277588, 2.579955816268921, -0.9903979897499084, 0.1011831983923912, -0.07334501296281815, -0.7915678024291992, -0.1538928747177124, -0.8539465069770813, -0.059009723365306854, 0.7925584316253662, -0.8560293912887573, 0.10532080382108688, 1.010806679725647, 0.2515988051891327, -0.5654637813568115, 0.12990890443325043, 0.4528270959854126, -0.5675534605979919, 1.4538425207138062, -1.556006908416748, 0.6999416947364807, 0.008798257447779179, -0.1357373297214508, -0.011520070023834705, -2.503976345062256, 0.028510816395282745, 1.3055014610290527, -0.1283775269985199, 1.306549072265625, 1.0820047855377197, -0.007073923014104366, -0.4086575210094452, -1.0462594032287598, 1.4539850950241089, 0.026797249913215637, -0.74635249376297, 0.7242556214332581, -0.034602899104356766, 0.0396086648106575, -1.617478370666504, -1.0180180072784424, -0.055422909557819366, -0.13082151114940643, -0.5146053433418274, -0.05124086141586304, 0.12622451782226562, 0.057311609387397766, 0.6548559665679932, 0.27327272295951843, 0.14894510805606842, -0.36574143171310425, -3.1682021617889404, 1.5379648208618164, -0.20717667043209076, -0.1667923927307129, -0.1186041608452797, -0.44730788469314575, 1.4725062847137451, -0.28375476598739624, 1.447723388671875, -4.277169704437256, 0.9083923697471619, 1.221411108970642, -0.2875072658061981, -1.1041955947875977, -1.0344022512435913, 1.1311330795288086, 0.672921895980835, 0.5836893916130066, -0.0797669067978859, -1.0687330961227417, -0.019271541386842728, 0.4808691143989563, 1.0691059827804565, 0.2158709615468979, 0.9034048914909363, -0.6654014587402344, -0.20554262399673462, -0.7348129153251648, -0.9604023098945618, 0.29472485184669495, 0.11179111152887344, -1.416259765625, -0.0017817140324041247, 1.3779428005218506, 0.4287046492099762, -0.4158101975917816, 0.8109965920448303, -0.826806366443634, -0.11097991466522217, -0.38379284739494324, -1.6303671598434448, -0.1627160757780075, -1.874774694442749, 0.5197078585624695, -4.93972110748291, 0.8614554405212402, 0.7553167939186096, -0.3097343146800995, -0.007664269767701626, -0.17859728634357452, -0.005716692190617323, -0.16231895983219147, 0.22667992115020752, 0.0003869088541250676, -0.19185414910316467, 1.1919444799423218, 0.7572901844978333, 0.6091196537017822, -0.863405704498291, 0.6594567894935608, -1.8377214670181274, 3.218688488006592, 0.29514726996421814, 0.7111262679100037, -2.003791332244873, 0.155475452542305, -0.08604235202074051, -0.4502846300601959, 0.14049333333969116, 0.24880333244800568, -0.9457592368125916, -1.0722662210464478, 2.5068087577819824, 0.1277746856212616, 0.6747869253158569, 0.4860423505306244, -0.9134392738342285, -0.2938404977321625, -0.37343910336494446, -0.02106069028377533, 0.09536638855934143, 0.5224261283874512, -1.1640931367874146, 0.30014097690582275, -1.8067443370819092, 0.08524066209793091, -0.1721612811088562, 1.4995955228805542, -0.31907951831817627, -0.4536917507648468, 0.9078156352043152, 0.32649049162864685, -2.5751264095306396, 0.5090765357017517, -0.8064313530921936, 0.11067022383213043, 0.8073680400848389, 0.26589474081993103, 0.3312699794769287, 0.006092945113778114, -0.1584734320640564, -0.4772232174873352, -0.6938526630401611, -1.794106364250183, -0.29267334938049316, 0.7793306112289429, -0.4582951068878174, -1.0807805061340332, 0.5954252481460571, -0.03342023119330406, 2.990623950958252, 0.11921301484107971, 0.9656535387039185, 0.35330653190612793, 2.1748714447021484, 1.5265538692474365, 0.37728118896484375, -0.11090470105409622, 0.05463508144021034, 0.9484958052635193, 2.5267255306243896, 0.7033030986785889, 0.0813012644648552, -4.743129730224609, 0.7264931797981262, -0.5477883219718933, -1.316657304763794, -1.7496893405914307, -1.0730831623077393, -1.9386838674545288, -0.48846590518951416, 1.7631556987762451, -0.5948270559310913, 2.494234085083008, 1.3256369829177856, 2.978379249572754, -0.054374612867832184, 1.192298173904419, -0.08005782216787338, 0.13491228222846985, -0.005193132441490889, 0.007647513411939144, 0.2962392270565033, -0.1959860920906067, -0.8503666520118713, -0.6217965483665466, 0.13212522864341736, 0.29038628935813904, 0.039336733520030975, -0.12775753438472748, 1.6814583539962769, 0.7241207361221313, -0.1810850203037262, 0.2329663634300232, 0.0032228806521743536, -1.7405465841293335, -0.31225982308387756, 0.3222994804382324, -0.18483836948871613, 0.1107989177107811, -0.40914735198020935, 1.852970004081726, 0.022668981924653053, -0.047780293971300125, -0.051480621099472046, -0.2951624095439911, 0.19794133305549622, 1.0081677436828613, 1.1309629678726196, 1.3590047359466553, -0.8974188566207886, -0.0160402599722147, -0.8283478617668152, 0.2925944924354553, 1.1136208772659302, -0.01747766323387623, 0.21160709857940674, 0.9637891054153442, -0.5163956880569458, -0.11105780303478241, -0.15714576840400696, 0.12637878954410553, -0.0002569771895650774, 2.7764506340026855, 0.03448469564318657, 0.534775972366333, 0.40436217188835144, -0.6524115800857544, 0.6644092202186584, 0.3709069788455963, 1.8016172647476196, -0.17087572813034058, -1.2807328701019287, -0.7976518273353577, -1.1761038303375244, 0.9607844352722168, 0.23784448206424713, 0.9012201428413391, 0.8696702122688293, 1.420851707458496, 0.0280750822275877, -0.5268419981002808, 0.08138050138950348, 0.5242365002632141, -0.9246371388435364, 0.13969573378562927, -0.15525726974010468, -0.354849636554718, 0.5894705653190613, -0.007722221314907074, -0.6784034967422485, -0.12306581437587738, 0.10632175207138062, -0.009521755389869213, -0.14888951182365417, 1.9655097723007202, 0.9527834057807922, -0.20437754690647125, -0.14399510622024536, -0.1883939653635025, 0.9278207421302795, 0.5889754295349121, 0.32729947566986084, -0.41194871068000793, 1.5762873888015747, -0.1377718448638916, 0.538911759853363, -0.7312125563621521, -0.205003559589386, 0.06166583299636841, -0.40017345547676086, 0.016719119623303413, 1.094265341758728, 0.9185682535171509, -1.6497745513916016, -1.0505046844482422, 0.12865696847438812, -0.06664534658193588, -2.2771034240722656, -0.13314507901668549, -0.83570796251297, 0.43248534202575684, -1.1541427373886108, -0.44723039865493774, 0.45410290360450745, 0.40374064445495605, 0.09698459506034851, 0.3816044330596924, 0.903304934501648, 0.4318125247955322, -0.8410135507583618, 2.098860740661621, 0.2054699957370758, 0.6563512682914734, -0.7513309121131897, 1.0411993265151978, -0.16279387474060059, 0.2675241231918335, -0.08105413615703583, 0.8407993912696838, -0.41702085733413696, 0.15533821284770966, -0.11692140251398087, 1.0467802286148071, -0.8313827514648438, 0.888992428779602, 1.0384601354599, 0.0061501627787947655, -0.08412052690982819, -0.49877703189849854, 0.5245868563652039, 0.3906765580177307, -0.20666663348674774, 2.807481527328491, 0.9121281504631042, 2.2374672889709473, -0.37695959210395813, -0.2396882027387619, 0.001720008789561689, 0.1271117627620697, -0.0031215210910886526, 0.6152448058128357, 0.5142484903335571, 0.1080315038561821, 0.2205994874238968, -0.369311660528183, -0.8821238875389099, -0.4970419406890869, -1.2335293292999268, -0.5316464304924011, -0.006806632969528437, 0.2530461549758911, -1.592833161354065, 0.16606825590133667, -1.8627984523773193, 0.7214028835296631, -0.7180983424186707, 0.5206968784332275, 0.32063668966293335, 0.9622251391410828, -1.1071282625198364, -0.5679073333740234, -0.3692849278450012, 0.16913679242134094, -0.04469272866845131, 0.6798537969589233, -0.6038066744804382, -0.511714518070221, 0.0037214765325188637, 0.8340131044387817, 0.10013474524021149, 1.8249250650405884, -0.3601241409778595, -0.7325111627578735, -0.012990033254027367, -0.8517282605171204, 0.03435494005680084, 0.29184526205062866, 0.016429690644145012, -0.5921396017074585, -0.036186765879392624, 1.1813651323318481, 0.27132946252822876, -0.9745807647705078, 0.08769338577985764, -0.409374862909317, -0.1316438466310501, -0.15555647015571594, 0.29185083508491516, 0.14711497724056244, 0.33725297451019287, -0.6875320672988892, -0.5379495620727539, -0.44313639402389526, 0.031638819724321365, -2.809983253479004, 0.9060572981834412, -0.17461510002613068, 0.685018002986908, -0.034236710518598557, 2.2855937480926514, -0.4487053453922272, 0.15751323103904724, -0.035930074751377106, -0.7597166895866394, 0.16409796476364136, -0.5258287191390991, 0.011040965095162392, -0.30737289786338806, 0.7338318824768066, -0.5069049596786499, -0.8453767895698547, -0.32462987303733826, 0.37636443972587585, 0.37575286626815796, -0.8106163740158081, -0.0022617769427597523, 0.18197527527809143, 0.4500938355922699, 0.38484901189804077, 0.02322089858353138, -0.47093290090560913, -0.05038262531161308, -0.7010518312454224, 0.38817518949508667, 0.2465323954820633, -0.19699111580848694, -0.20663276314735413, 0.009737764485180378, 0.3105565309524536, -0.0026890155859291553, -0.19353017210960388, 2.1180689334869385, -1.3122501373291016, 2.4317469596862793, 0.4833139479160309, 0.07475713640451431, 0.512458086013794, -0.9672854542732239, 0.28880080580711365, -0.3270546793937683, 0.7177969813346863, -0.044101208448410034, -0.06778937578201294, -0.18557006120681763, -0.6335694193840027, -0.1237119510769844, -2.139950752258301, -0.5268901586532593, 0.12973274290561676, 0.01568988524377346, -0.03962118551135063, -0.666254460811615, -0.5593846440315247, 1.2594586610794067, -0.1694994568824768, -0.07238317281007767, 0.28360164165496826, -0.9288293123245239, -2.6383559703826904, -2.584399938583374, 1.2873417139053345, 2.4052820205688477, 0.003876449540257454, 0.8112639784812927, 0.02004219777882099, 0.854426383972168, 0.46857893466949463, -0.7647743225097656, 0.1707817018032074, 0.8956314325332642, 1.5597163438796997, -0.7168137431144714, -0.05931463837623596, -0.06223217770457268, -1.752922773361206, -0.039950594305992126, -0.24304725229740143, -0.2996494174003601, -1.8555853366851807, 0.01943536475300789, 1.8537330627441406, -0.31281864643096924, -0.19444487988948822, -0.17931140959262848, -1.6038342714309692, 0.07296818494796753, 0.12589824199676514, 0.046358589082956314, -0.04273513704538345, -0.13431987166404724, 0.5542494654655457, -1.796830415725708, -0.8436580300331116, 0.25878241658210754, 0.6257693767547607, -0.6449502110481262, 0.03468897193670273, -0.04093383625149727, -0.19354557991027832, -0.01464847568422556, 0.0029579377733170986, -0.3295661509037018, 0.4110559821128845, -0.7467337250709534, -0.38321229815483093, 0.2500331699848175, -0.09553491324186325, -0.302158385515213, -0.3741866648197174, 0.7080057859420776, 2.5362391471862793, 0.4082725942134857, 0.9393131136894226, 2.4116010665893555, -1.9724212884902954, 0.849178671836853, 1.5015876293182373, 0.5511072278022766, -0.4750949740409851, 0.10684613138437271, 5.414103031158447, 0.0003174827725160867, 0.44520455598831177, 0.008076806552708149, -0.04255970939993858, -2.093501567840576, 0.06378445774316788, -0.004936954006552696, -1.3208974599838257, 0.7895304560661316, 0.6607038974761963, 0.44201162457466125, -1.149006724357605, -0.41245612502098083, 0.3466653525829315, -0.3455750048160553, 0.015084868296980858, 1.4878699779510498, -0.024178139865398407, -0.30930933356285095, 0.8216577172279358, -0.40884920954704285, 0.11248281598091125, 0.003143065143376589, 0.03995668143033981, 0.9769070744514465, 0.21962350606918335, -0.010793735273182392, -0.2336280643939972, -0.7867406606674194, 0.041886381804943085, -0.035390980541706085, 0.7303037047386169, -0.41369956731796265, -0.4829464256763458, -1.090084433555603, 0.03511737287044525, -2.2344555854797363, 0.020776011049747467, -1.3593443632125854, -0.01760634407401085, -0.006266721989959478, -0.45598840713500977, 0.0038314249832183123, -2.178772449493408, -0.26928213238716125, -3.5428977012634277, 0.5107758045196533, 0.06440994888544083, -0.881273627281189, 0.3269776999950409, -0.2583411633968353, -0.11455892026424408, -0.23982486128807068, -0.04441842436790466, -0.0690445750951767, -0.4900597929954529, -0.7366048097610474, -1.0587869882583618, 0.01973617821931839, -0.17928001284599304, 1.1012200117111206, -3.2312867641448975, 0.32508331537246704, 0.2740817666053772, 1.960282564163208, 0.34945157170295715, 0.6248695254325867, 0.18854399025440216, -2.4649548530578613, 0.3019247055053711, -0.004208714701235294, -0.03860176354646683, -1.5339362621307373, -1.02634596824646, 0.44102227687835693, 2.148082971572876, -0.6846531629562378, -1.2078033685684204, 0.14013101160526276, -0.10018686205148697, 0.18907956779003143, -2.649207592010498, -0.1956937462091446, 1.1832331418991089, -0.08641905337572098, 0.5427008867263794, 0.11534330248832703, -0.49635961651802063, -0.3789001703262329, 0.20677411556243896, -1.4753779172897339, 0.3151415288448334, -0.5194386839866638, -0.11123032122850418, 0.25688204169273376, -0.35734471678733826, -0.7149546146392822, 0.776890754699707, 0.608415424823761, 0.5814372897148132, 0.23432855308055878, 0.05174053832888603, -1.5482637882232666, -0.9650864601135254, -3.1362857818603516, -0.7138179540634155, 1.0721017122268677, 0.30524328351020813, 0.0046712965704500675, 0.0011397207854315639, 0.5321612358093262, 0.5395528078079224, -0.22436250746250153, -1.136676549911499, -0.5441418886184692, 0.4438963830471039, -0.7734350562095642, 0.0007359512965194881, 0.03470919281244278, -0.8296847343444824, 0.6403467059135437, -0.61305832862854, -1.0754245519638062, -1.1736888885498047, -0.005372507497668266, 0.5506945252418518, 0.09288571029901505, -1.9047287702560425, 0.9236338138580322, -1.7487907409667969, 0.12574878334999084, 0.3280671238899231, 0.1168486624956131, 0.18491537868976593, 1.4134547710418701, 0.1378297060728073, -1.530530571937561, 0.1620797961950302, 1.1204993724822998, -0.24246922135353088, -0.9573697447776794, -0.03981686383485794, 0.31527528166770935, -1.308641791343689, 0.4531046152114868, -0.6214364767074585, -0.2107950896024704, 2.59108304977417, -1.2162048816680908, -0.31639060378074646, 0.7583435773849487, -0.02993791364133358, -0.06343064457178116, -0.0639609694480896, -0.10437386482954025, 0.8168058395385742, 0.16093577444553375, 0.44387680292129517, -0.4211527407169342, 0.09469395875930786, 0.08421796560287476, 0.5976417660713196, -0.9480053782463074, 0.1365644931793213, -0.044155679643154144, 0.343055784702301, -0.46329522132873535, 1.3176062107086182, -0.0475936084985733, 2.2817611694335938, 0.020832976326346397, 0.18445511162281036, -0.5742494463920593, -0.5466849207878113, 0.620665431022644, -0.49759894609451294, -2.2462704181671143, -1.421347975730896, 0.7226017713546753, 0.4539436995983124, -0.6211617588996887, 1.745398759841919, -0.9042363166809082, 0.1959058791399002, -0.011377319693565369, 0.9853657484054565, 0.01597355306148529, -0.02506905049085617, -2.762907028198242, 0.23481309413909912, 1.2371388673782349, -1.2965410947799683, -0.48031139373779297, -0.6574196219444275, 0.04836327210068703, -0.29306697845458984, -0.029662923887372017, 0.0878247395157814, -0.12523208558559418, 1.7280508279800415, -0.18070165812969208, 1.2273433208465576, -0.1677727997303009, 1.1824414730072021, 1.3030883073806763, 0.4395042955875397, -0.35355421900749207, 0.7594102025032043, -0.033928677439689636, 0.2376464307308197, -0.13529904186725616, 0.2603180706501007, -0.05570174381136894, -0.859607458114624, -0.3573538064956665, -0.14504088461399078, -0.6149469017982483, -0.04474417120218277, 0.0009634225280024111, -0.002425817074254155, -0.5878669023513794, 0.10110793262720108, -0.40850961208343506, 0.10590890794992447, 1.0509843826293945, 0.6191730499267578, 2.292456865310669, 0.006930049508810043, -0.11165929585695267, 0.7458001971244812, 1.2986657619476318, -0.4486711621284485, 0.7729618549346924, 0.006850899662822485, 0.32013699412345886, -0.8266110420227051, -0.5515695214271545, 0.4591294229030609, -2.5727546215057373, -1.7900639772415161, -0.3672983944416046, -0.598908543586731, -0.060775045305490494, -0.11323206126689911, -0.3712522089481354, -0.17780309915542603, -0.14090785384178162, 2.6919469833374023, -0.9199605584144592, -0.3555089235305786, -0.38676953315734863, 2.0283076763153076, -0.2637515962123871, 0.20729771256446838, -0.8985819220542908, -0.07919133454561234, 1.0827209949493408, -0.04942670091986656, -0.789613664150238, -0.044720232486724854, -0.5223643183708191, 3.6505045890808105, 0.4299227297306061, -0.040992747992277145, -0.07755817472934723, 0.4846937358379364, -0.4063735902309418, 0.058795321732759476, 0.4149649143218994, -0.057366494089365005, 0.4616716206073761, -0.09230340272188187, 0.007232471369206905, 0.08363243192434311, 0.19875165820121765, -0.615382194519043, -0.0144511042162776, -0.20162251591682434, 0.18499353528022766, 0.9250285625457764, -0.7469815015792847, -0.8416722416877747, -1.6705987453460693, 0.3727234899997711, -1.7468124628067017, 0.38723716139793396, 2.125523805618286, -0.21138910949230194, 2.132341146469116, 2.1504476070404053, -1.1143527030944824, -9.827712059020996, -0.003576605347916484, 0.5289015769958496, -0.23956415057182312, 0.11331694573163986, -0.7541103959083557, 0.07443328201770782, -0.0010697203688323498, 0.05208761245012283, -3.0378267765045166, -1.1121361255645752, 0.9727576971054077, -0.42091113328933716, -1.3861244916915894, 0.027075543999671936, -2.9357709884643555, 0.12682348489761353, 1.2296125888824463, -0.1272256225347519, -1.5388693809509277, 0.2843768000602722, -0.053467605262994766, 0.8503128290176392, -1.7498462200164795, 0.204242542386055, 0.5339095592498779, -0.09186531603336334, 0.3101764917373657, 0.07310481369495392, 1.156192660331726, 0.2756437361240387, 0.14317801594734192, -1.1872278451919556, 2.0502302646636963, 2.8997411727905273, -0.9196190237998962, -0.2812771797180176, 0.23783761262893677, 0.06536925584077835, 0.1403128206729889, -2.120802164077759, -0.04737816005945206, -0.6025436520576477, 0.011040503159165382, -0.005152399651706219, -0.4866737127304077, -0.9233987927436829, 1.2719172239303589, 0.001271053566597402, 0.5477480292320251, -0.49440333247184753, 0.00520644336938858, 1.7434873580932617, -0.6193437576293945, 0.12659971415996552, 1.3308650255203247, -1.5217386484146118, 2.221694231033325, -0.7082754373550415, -0.27253884077072144, -0.09426933526992798, -0.06833739578723907, -0.22072143852710724, 0.8054545521736145, 0.015586325898766518, 1.7727841138839722, -0.4224480092525482, 0.6847223043441772, -1.1730504035949707, 0.5101890563964844, -0.8499642610549927, -0.824954092502594, -0.32457372546195984, 0.8189838528633118, -0.016179369762539864, -0.01791253127157688, 1.9178516864776611, 0.016067449003458023, -0.5341575741767883, -2.1888718605041504, 0.8202418684959412, -0.010308622382581234, 0.14455640316009521, 0.9285277128219604, 0.7637400031089783, -0.024775756523013115, -0.16447697579860687, 0.33169877529144287, -0.17137494683265686, -0.3415982723236084, 2.948530673980713, -0.13512930274009705, 0.6836870312690735, -0.01750108413398266, -0.016410835087299347, -1.5320255756378174, 0.569984495639801, -0.8076850771903992, 0.12776674330234528, -0.8240286707878113, -0.698115348815918, 0.1594749093055725, 0.17997314035892487, 1.1340779066085815, -0.7614743709564209, -0.508338451385498, -0.0350305438041687, 0.33689790964126587, 0.09845954179763794, 1.120377779006958, 0.7605096101760864, -0.0711347758769989, 0.008651774376630783, 0.05683925747871399, 1.8460837602615356, 0.14099496603012085, 0.12308673560619354, -0.7380820512771606, 0.10171645134687424, -1.059312105178833, -2.20672607421875, -0.19054900109767914, -0.5556111931800842, -2.4908058643341064, -1.3536409139633179, -0.356484979391098, 0.5556371808052063, -0.4958896338939667, -1.3866685628890991, 0.1737663596868515, -1.0575751066207886, 0.07794277369976044, -0.7103385329246521, -0.27740049362182617, -0.3364883363246918, 0.1363028585910797, 0.23345443606376648, 0.15857858955860138, 0.3973025679588318, 0.6870167255401611, 3.007568836212158, 0.07785829901695251, -0.2017841637134552, 0.010405860841274261, 0.8161576986312866, -0.028333915397524834, -0.10487983375787735, -0.21576818823814392, -0.3068694472312927, 3.2441844940185547, -0.9691272974014282, 0.2962741553783417, 0.5697623491287231, -1.7862050533294678, -1.5441817045211792, 0.06680083274841309, 1.6698054075241089, -0.005520931910723448, 0.7174211740493774, -0.05399201437830925, -0.16939406096935272, 0.5163777470588684, 0.5059467554092407, 1.0364187955856323, -0.34473881125450134, -0.02561086229979992, -0.7319482564926147, -1.0767557621002197, 0.5735111832618713, 0.03277014568448067, -1.5866901874542236, 0.06293540447950363, -0.5975933074951172, 3.426549196243286, -0.6337510943412781, 0.595197856426239, -1.041611671447754, -0.6446970105171204, -0.15275715291500092, -0.5965753793716431, -0.028722768649458885, 0.10224800556898117, 0.37097030878067017, -0.7078256011009216, -0.08486970514059067, -0.40865686535835266, 0.08029132336378098, 0.14997069537639618, 0.14789266884326935, 0.4787665903568268, -0.6246798634529114, 0.26450443267822266, 0.37188923358917236, -0.25241151452064514, -1.4704886674880981, -0.7825335264205933, -0.08757896721363068, 0.0484231673181057, -0.2664740979671478, 0.19225025177001953, -0.1299746334552765, 0.03298920765519142, -0.17928335070610046, 0.8336135149002075, 0.0472712479531765, -0.16058284044265747, -0.2727909982204437, 0.28744861483573914, 1.985996961593628, 0.21504798531532288, 0.0072014424949884415, -0.1259431093931198, -0.2587280571460724, -2.4747278690338135, -0.18876105546951294, 0.3746219575405121, -0.36885014176368713, -0.11583640426397324, 0.1724901795387268, 1.4872673749923706, -0.08995585143566132, -1.4922924041748047, -0.19628368318080902, 0.30283644795417786, 0.01321658305823803, -0.24201907217502594, -0.5642647743225098, 1.640823483467102, 0.9508704543113708, 0.643441379070282, 0.2153245210647583, 2.1633105278015137, -7.458907604217529, -2.33708119392395, -0.3963169455528259, 1.17144775390625, -3.2996251583099365, -1.325485110282898, 0.9171493649482727, -0.23458363115787506, 0.23142042756080627, -1.3264508247375488, 0.020456766709685326, -0.06480755656957626, -0.0506228432059288, -0.41118091344833374, -0.3229001760482788, 0.3381024897098541, 0.22200044989585876, 0.41082969307899475, 0.11422790586948395, 0.09614909440279007, 0.11739557236433029, 0.21108272671699524, 1.5325708389282227, -0.010480135679244995, 0.28162747621536255, -0.36988574266433716, -0.05522187426686287, -1.4445250034332275, 0.36163756251335144, -0.010345082730054855, -1.0228480100631714, -0.17046266794204712, 0.024938181042671204, -0.29342547059059143, 0.7412015199661255, -0.6102198362350464, -0.637144923210144, -0.23767292499542236, 0.24361376464366913, -0.03664669767022133, -0.021432960405945778, -1.4126137495040894, 1.046815276145935, 0.3864021897315979, -0.15232759714126587, 0.3403152525424957, -0.004409130662679672, -0.9611712098121643, -2.2641825675964355, 0.011286460794508457, 0.48549363017082214, -0.261229932308197, -0.12439985573291779, -0.3957989513874054, -0.8966861963272095, -2.1042885780334473, -0.26514387130737305, -0.7821671366691589, 0.25183913111686707, -0.10702789574861526, 0.07795332372188568, -1.4385443925857544, 0.2167743742465973, -0.24612869322299957, -1.310533881187439, 0.13084803521633148, 0.03782307356595993, -0.5305425524711609, -0.26712262630462646, 1.0779893398284912, 0.3776756823062897, 0.09908740967512131, -0.12867219746112823, 0.9016380310058594, 0.11811898648738861, -0.9114468693733215, 0.23205621540546417, 0.41669315099716187, 1.0138188600540161, 0.21990785002708435, 1.8793636560440063, 1.1475034952163696, -1.3894963264465332, 1.3680726289749146, 0.9229781031608582, 0.1796800196170807, 0.8167523145675659, -0.34242576360702515, -0.03642522916197777, 1.4063117504119873, -1.0874824523925781, -0.20856305956840515, -0.7753999829292297, -0.008444498293101788, -1.4912225008010864, 0.20794826745986938, -0.12355542182922363, -1.4599653482437134, 1.209232211112976, -0.9438985586166382, 0.61619633436203, -0.022992892190814018, -0.3029169738292694, -0.024291880428791046, 0.44963452219963074, 0.39211463928222656, -0.5448532104492188, 0.6327784061431885, -0.012400612235069275, -1.7954145669937134, 1.4731435775756836, 0.0008536085952073336, -0.5345660448074341, 1.024601936340332, 0.5871466398239136, -0.31360456347465515, 1.3717058897018433, -0.1498943418264389, 0.08641867339611053, 0.6446731090545654, -1.778455376625061, -1.1873754262924194, -0.39908549189567566, -0.07565294206142426, 1.6147582530975342, 0.4895961880683899, 0.5832859873771667, 2.652332305908203, 2.688849449157715, -0.17886918783187866, 0.23860467970371246, 2.075824737548828, 0.80312180519104, -0.15787363052368164, 0.8396360874176025, -0.34498146176338196, -1.0027987957000732, -0.15183125436306, 1.6738319396972656, 0.12496839463710785, 0.002919998485594988, -6.378314018249512, 0.16733744740486145, 0.24723772704601288, 0.01234834548085928, -0.5445943474769592, 0.04408370330929756, -0.0969310849905014, -0.1805226057767868, 0.3294968008995056, 0.5857334136962891, 0.41445988416671753, 0.4406537711620331, -1.0595146417617798, 1.8762298822402954, 0.9725451469421387, -0.7694253921508789, -0.5863044261932373, 0.6958464980125427, 1.5846388339996338, 0.5930052995681763, 0.7014906406402588, -0.025130923837423325, 0.10198938846588135, 1.0148557424545288, 0.5634607076644897, 0.2685222029685974, 0.3410424292087555, -0.22396546602249146, -2.3611176013946533, 0.044033233076334, -0.9816373586654663, -0.27463671565055847, -0.26265835762023926, 0.8876115083694458, -0.13435126841068268, -0.09214471280574799, -0.5751344561576843, 0.15939246118068695, 0.04107894003391266, -0.01061648316681385, 0.28041091561317444, -0.471859872341156, -0.1261427104473114, 0.09242216497659683, 0.16161032021045685, 0.4884089529514313, 0.01082698255777359, -0.01454173494130373, 0.053412698209285736, -1.9386426210403442, 0.29604554176330566, -0.9195908308029175, -0.35181909799575806, -0.1300879269838333, -0.01266234926879406, -0.08224760740995407, -1.1910536289215088, -0.05188760533928871, -1.51175057888031, 1.9686030149459839, -0.5260090827941895, -0.1819855272769928, 0.23083345592021942, -0.025336168706417084, -0.14909476041793823, 0.1327979862689972, 0.2867854833602905, 1.3546736240386963, -0.14643032848834991, -0.027486683800816536, -0.7366273999214172, 0.45758503675460815, -1.1524693965911865, -0.028402065858244896, 0.9536383748054504, 0.15488047897815704, -0.6244227886199951, -1.5801738500595093, -0.21381045877933502, -0.13718043267726898, -2.40893816947937, 1.4119371175765991, 0.5118341445922852, 0.12347836792469025, -0.19525498151779175, -0.3115171492099762, 1.2193037271499634, -0.11742209643125534, 0.23282437026500702, -0.3264094591140747, -1.3625826835632324, 0.29949867725372314, 0.4885934293270111, -0.5384538769721985, -1.1184556484222412, -0.4348243772983551, -1.044150471687317, -0.08914391696453094, 0.845615804195404, 0.1407138556241989, -0.33354103565216064, -0.6875069737434387, -1.115317463874817, 0.4879100024700165, -1.0267868041992188, 0.32345935702323914, 1.6982338428497314, 0.9173251390457153, 1.828350305557251, -0.1818610429763794, -0.13296112418174744, 1.2918574810028076, -0.4744950830936432, -1.6910953521728516, 0.010192743502557278, 0.508162796497345, 0.09517146646976471, 0.2687744200229645, 0.08173937350511551, 0.050158511847257614, 0.626813530921936, -0.2396591305732727, 0.2577439248561859, -0.11884079873561859, -0.33537474274635315, -0.1764754205942154, 0.6634186506271362, 0.5613085031509399, 0.13666114211082458, -0.20068638026714325, -1.8812192678451538, -0.21239285171031952, 0.4329151213169098, 1.6643208265304565, -0.16343992948532104, 0.11003671586513519, -0.08957435190677643, 0.41856715083122253, 0.5742046236991882, 3.084470510482788, 0.3520551323890686, 0.010072709992527962, -0.31853213906288147, -0.7411087155342102, 0.6442457437515259, 0.09053760021924973, 0.657413899898529, 0.6723331212997437, 0.9523437023162842, -2.460115432739258, 0.3344374895095825, 1.3508071899414062, -0.17310041189193726, -0.08849190175533295, -0.016349412500858307, 0.8467274904251099, 0.7975820899009705, -2.522136688232422, -1.758476972579956, 0.601673424243927, -1.1969215869903564, -0.029043402522802353, -0.6535497307777405, 0.8233502507209778, 0.6298006772994995, -0.03682290017604828, 0.10008785128593445, 0.06391021609306335, -0.3285371959209442, 0.6624210476875305, 1.9685698747634888, 2.209099769592285, -0.20959033071994781, 0.020916245877742767, -0.5656862854957581, 1.2212319374084473, 2.8839948177337646, 0.04059072956442833, -0.06968430429697037, 0.3639678657054901, 0.7938898801803589, -0.28629299998283386, -0.2560333013534546, 2.4945616722106934, -0.8963079452514648, -0.14913934469223022, 0.04649225249886513, 0.7289376258850098, -1.7405632734298706, -0.1429819017648697, 0.011621142737567425, -0.07628317177295685, -3.689229726791382, 0.3774482309818268, -3.5573697090148926, -0.3512252867221832, -0.5336157083511353, -0.6061088442802429, 0.6840387582778931, -0.011799152940511703, -1.5066776275634766, 1.223325490951538, -0.007102746516466141, 1.0573559999465942, -0.6669002175331116, 0.6054031252861023, -0.008804725483059883, -0.5445829629898071, -0.8978241086006165, 0.3297029733657837, -0.6694429516792297, -0.6412843465805054, -0.02327042445540428, 0.5172178745269775, 1.3684682846069336, -0.01776600442826748, -0.2667977511882782, -0.7440311908721924, 5.301513195037842, -0.15017323195934296, 2.2831802368164062, -0.6658942699432373, 0.42991340160369873, 0.31105709075927734, -0.015724774450063705, -0.34435853362083435, -0.08900010585784912, 0.119858518242836, 0.04121256619691849, 0.07388496398925781, -0.5180644392967224, -0.3884970545768738, 0.10453864187002182, 2.0472285747528076, 0.07715590298175812, -0.3028006851673126, 0.1206754744052887, -0.6402174234390259, 1.3012620210647583, 0.14670512080192566, 1.860986351966858, 2.3772313594818115, 2.9021646976470947, 0.28002482652664185, -0.8518307209014893, -0.19496046006679535, 0.29565688967704773, -0.44173741340637207, 1.0881366729736328, 0.028904197737574577, -0.8038504123687744, -0.1075320914387703, 0.024579769000411034, -0.03088677115738392, -0.2511449456214905, 0.07421804219484329, -0.014707749709486961, -0.40278545022010803, 0.2013486623764038, 0.17111772298812866, -2.4200172424316406, -0.09983737021684647, 0.3313298523426056, -1.4331375360488892, -0.22882534563541412, 0.03487393260002136, -0.8420165181159973, 0.026719557121396065, -0.3434675335884094, -0.3937641680240631, -0.0736321285367012, -0.13651438057422638, -0.0002461695112287998, 0.5226688981056213, 0.7767369747161865, -0.209122434258461, 0.34085455536842346, 0.9910353422164917, 0.05190161615610123, 0.5373957753181458, 0.19918937981128693, -0.40182316303253174, 1.2854514122009277, 0.1168549507856369, -0.03509678691625595, 0.27799755334854126, 0.04112894833087921, -0.2754655182361603, -0.044665683060884476, -0.3904767632484436, -0.24760733544826508, 0.7159884572029114, 0.7114214897155762, 1.22760009765625, 0.0035765566863119602, 0.8123443126678467, 0.6513347625732422, 0.08494055271148682, -0.5507944226264954, -0.1588572859764099, -1.3444265127182007, 0.10505815595388412, -0.14956673979759216, -0.01782798022031784], [0.19430407881736755, -0.8622354865074158, 0.04166517034173012, -0.13549037277698517, 0.6462294459342957, -0.7741938233375549, -0.8613362908363342, -0.20229396224021912, -0.05086340755224228, 0.9380919337272644, -0.5496560335159302, 0.15765142440795898, 1.0528146028518677, -2.3794641494750977, -0.39773914217948914, 0.033139705657958984, -0.49005916714668274, -0.36841732263565063, 1.4804586172103882, 0.07068696618080139, -0.5595402121543884, -0.04219057783484459, -0.8099615573883057, 0.4746910333633423, 0.3817170560359955, 0.8393611311912537, -0.41289255023002625, 0.34256717562675476, -1.115785002708435, 0.25092214345932007, 0.19676543772220612, -0.5333849787712097, 0.2339503914117813, -0.009519681334495544, 0.7598795890808105, -0.04497655853629112, 0.4570031762123108, 0.8150545954704285, 0.01348172314465046, 1.9258371591567993, 0.3522281348705292, -0.08506402373313904, -1.7156894207000732, 0.03182283788919449, 1.3647691011428833, -1.8598806858062744, -0.034109748899936676, 1.2079721689224243, -0.441224068403244, -0.3422102630138397, 1.6079038381576538, -0.2408759742975235, -1.7612662315368652, -0.9891804456710815, -0.04195528104901314, -1.4979913234710693, 0.8626741170883179, -0.035223376005887985, -0.6334882378578186, 0.4484838843345642, -0.499933123588562, 0.6876621842384338, -1.3992608785629272, 0.23164398968219757, 0.06489821523427963, 0.045590441673994064, 0.028323987498879433, -4.222229480743408, 0.1731320023536682, -0.19553448259830475, -1.4998241662979126, -0.09803269058465958, -3.005247116088867, -1.336301326751709, -1.0693553686141968, 0.06451242417097092, -0.6867907047271729, -0.23251892626285553, 0.019473126158118248, 0.05660014599561691, -1.2091161012649536, -0.82066810131073, -0.22155767679214478, 0.2600230276584625, -0.7355273365974426, -0.029728615656495094, 0.042992234230041504, -0.5109509229660034, -0.21212932467460632, 0.10533225536346436, -0.08418715745210648, -0.2755284011363983, -0.132334902882576, 0.18520711362361908, 0.2385149598121643, -0.5160650014877319, -0.08808775246143341, -0.1566658914089203, -0.5015583634376526, 0.003834755392745137, 0.27177199721336365, 1.484142780303955, -0.44360774755477905, -1.1967897415161133, 0.17988848686218262, -0.10155487060546875, 0.8501284718513489, 1.3916441202163696, 0.0040952847339212894, 0.15126681327819824, -0.014540547505021095, 1.179624319076538, -2.312840461730957, 0.17363131046295166, 0.4854416847229004, 0.5361019372940063, 0.21673999726772308, 0.4215594530105591, -0.009654014371335506, 1.1630069017410278, -0.43070462346076965, -0.28146418929100037, 0.0023087444715201855, -0.11254563182592392, 1.720501184463501, 0.0660543367266655, 0.2991808354854584, 0.24284513294696808, -0.006023393012583256, 0.5978551506996155, -0.009823862463235855, 0.007822252810001373, 1.0450830459594727, -0.969113826751709, 0.12644633650779724, -0.29320332407951355, 0.030908210203051567, -0.2894306778907776, -0.01864212192595005, 0.36783310770988464, -0.08258156478404999, -0.2316744029521942, 0.6180039644241333, 1.2541301250457764, -4.451523780822754, -0.3049074411392212, -0.7461734414100647, 0.7782947421073914, -0.20322440564632416, -0.2797050178050995, -1.2396090030670166, -0.3238997757434845, -0.7416277527809143, 0.16172215342521667, -3.6179041862487793, 1.4277050495147705, -0.39042168855667114, -0.23767156898975372, -0.37108129262924194, 1.2914962768554688, -0.7721853256225586, -0.4017750024795532, -0.06489363312721252, 0.467119425535202, -0.5211169719696045, -0.46636682748794556, -0.21951483190059662, -0.061392586678266525, -2.298877239227295, -0.9447176456451416, 1.7327206134796143, -0.0999809131026268, -1.6166534423828125, 0.49225059151649475, 2.4350192546844482, -0.009536334313452244, 0.23014840483665466, -0.09535031765699387, -1.3838056325912476, 0.9493435025215149, -0.10089771449565887, -1.054388165473938, 1.1725399494171143, -0.6060717105865479, 0.12211667746305466, 0.17432138323783875, 0.0021029463969171047, 0.013100402429699898, 0.2650793492794037, 0.011405187658965588, 1.3197400569915771, 0.30932074785232544, 0.14327962696552277, 0.19322070479393005, -0.09018567204475403, -0.14271174371242523, -0.12259215116500854, 0.40802937746047974, 0.05594383552670479, -1.191152572631836, 0.2542618215084076, 0.01926715485751629, 0.017753614112734795, -0.3309680223464966, 1.099936842918396, 0.05812659114599228, 2.7193706035614014, -0.4951528310775757, -2.8178956508636475, 1.7132623195648193, 0.42279136180877686, -0.2260066121816635, -0.06652550399303436, 0.23378831148147583, 0.9052774310112, 0.36880746483802795, -0.9049894213676453, -5.537962913513184, -1.6079217195510864, 0.6165956258773804, 0.5015114545822144, -0.4095442593097687, 0.6977199912071228, -0.646441638469696, -0.5227880477905273, -1.3537037372589111, 0.28999558091163635, 1.0378538370132446, -5.2452006340026855, 0.5581417679786682, 0.012934280559420586, -0.15268972516059875, -0.013074235990643501, -0.44653576612472534, -0.03606686741113663, -0.9586634039878845, -0.12636233866214752, -0.15901540219783783, -0.15439023077487946, -0.029344482347369194, -0.07464378327131271, -0.1137324720621109, -0.2813788652420044, 0.7394714951515198, 0.10053015500307083, -0.03760822117328644, -0.13551642000675201, 0.24377602338790894, -0.07697614282369614, -0.9062257409095764, 0.019888609647750854, 0.35516560077667236, -1.9266210794448853, 0.07027900964021683, 0.6255910396575928, 0.34325602650642395, -0.411907434463501, 0.39517974853515625, -0.1403023600578308, 0.13377758860588074, -0.2870500087738037, -1.3988012075424194, 0.8040221929550171, -0.9357802867889404, -0.7547286748886108, 0.020826170220971107, -1.6764748096466064, 0.16986183822155, -0.1923823356628418, 1.1949315071105957, 0.22819940745830536, -0.07109968364238739, 0.5145779252052307, 0.05741757154464722, -0.7737460732460022, 0.31107527017593384, 0.18130701780319214, 1.5045305490493774, 0.20496289432048798, -0.2804397642612457, 0.5937445163726807, 1.2048603296279907, 0.5570937991142273, 0.2348850816488266, -0.12085738033056259, 0.22131995856761932, 0.39614179730415344, -0.15533939003944397, -0.11387420445680618, 1.0631475448608398, -0.05590863153338432, 0.2545560896396637, -0.13279403746128082, -0.10761790722608566, 0.11397190392017365, -1.9636660814285278, 0.6630289554595947, 1.0048555135726929, 0.8839462995529175, 0.007143605966120958, 0.2996634542942047, 0.46516329050064087, 0.6094180345535278, 0.17475348711013794, 0.6583224534988403, 0.5311589241027832, 0.9847742915153503, -0.26804739236831665, -1.0856963396072388, -0.23707343637943268, -0.8030978441238403, 0.031575679779052734, 0.18109136819839478, -1.7146226167678833, 1.2609493732452393, -0.9188140630722046, -0.25142166018486023, 0.28565388917922974, -0.013125642202794552, 0.282941997051239, 0.2650940716266632, 0.030621735379099846, -0.49597102403640747, 0.1940782219171524, -1.2372887134552002, -0.8088554739952087, 0.881617546081543, 0.1505313515663147, 0.2962406277656555, 0.4545604884624481, -0.114695243537426, 0.12859775125980377, 0.057709403336048126, -4.834292667510454e-06, 0.9816009402275085, 0.23377813398838043, 0.0026491773314774036, 1.1911447048187256, -0.008404863998293877, -1.433282732963562, -0.1593504399061203, 1.313806414604187, -1.0876731872558594, -1.124799370765686, 2.2504982948303223, 0.029982270672917366, -0.265080988407135, -0.15416136384010315, 0.5141326189041138, 1.5924279689788818, -0.019361233338713646, 0.05498174950480461, 0.322765052318573, -0.45423436164855957, -0.3769342303276062, 0.5592033267021179, -2.3723514080047607, 1.24275803565979, -0.05002894997596741, -0.09842287003993988, -0.48856598138809204, 1.734036922454834, -2.0499603748321533, -0.02095717005431652, -1.529639482498169, -0.8892934322357178, 1.0283688306808472, -0.15409357845783234, -0.06979496031999588, -1.253023386001587, 0.04579082131385803, -0.8670328259468079, -0.3175749182701111, 0.03597954660654068, -0.13577821850776672, 0.06528840214014053, -0.4610920250415802, 0.01426960900425911, 3.111257553100586, -0.3466905355453491, -0.07575641572475433, -0.5175800323486328, 1.3661696910858154, 0.6563077569007874, 1.359741449356079, 0.8532177209854126, -0.15341463685035706, 0.011000305414199829, 0.3849347233772278, -0.1125393882393837, -1.1729655265808105, -1.5652879476547241, 0.6090859770774841, -3.0910186767578125, 0.4978857636451721, -0.07494518905878067, -0.5600501894950867, 1.5516358613967896, 0.04079142585396767, -1.4959206581115723, 0.031499527394771576, 0.005388482939451933, 0.05906793847680092, 0.3804249167442322, 1.3260101079940796, 0.02330336533486843, 0.6630577445030212, 0.35994336009025574, -1.301712989807129, 2.655787944793701, -0.13080880045890808, -0.3040499985218048, -0.013290933333337307, -1.037389874458313, -0.5593984127044678, -0.030603643506765366, 0.12901872396469116, 0.03521573916077614, 0.008992341347038746, 0.6558836102485657, 0.0614786297082901, -0.023151881992816925, 0.5245696306228638, -0.3903611898422241, -1.2769721746444702, 0.8563485741615295, -0.36130884289741516, 0.5383440852165222, 0.11415961384773254, 0.3255760967731476, -0.3815986216068268, 0.23448514938354492, 0.1997043341398239, -0.11942411959171295, -4.527628421783447, -0.005792404059320688, 0.7612952589988708, 0.027436938136816025, 0.8067952990531921, 0.018639257177710533, 0.1901865303516388, 0.21103940904140472, -0.5430418252944946, 0.5730202198028564, -0.38437357544898987, -2.4357893466949463, 0.9637463092803955, 0.08705668896436691, 0.060233090072870255, 0.1306600123643875, 0.01598977856338024, -0.40584030747413635, -0.072660893201828, -1.7742775678634644, 0.026657545939087868, -0.3141278326511383, -0.11245571076869965, -0.33020439743995667, -0.08736252784729004, -0.11102757602930069, -0.4230528473854065, 0.909026563167572, 2.630983352661133, -0.17589659988880157, -1.482211947441101, -0.1629546582698822, -1.173537254333496, -0.5479282140731812, -1.0047186613082886, 0.0009030958754010499, 0.29929810762405396, 1.1048080921173096, 0.2754204273223877, 0.07747545838356018, -1.2009832859039307, 0.8865511417388916, -0.6940703392028809, -0.8194190859794617, -0.4525012969970703, 0.06057159975171089, -1.9150569438934326, 0.2637229859828949, -0.1709238886833191, -0.33913707733154297, -0.006358427461236715, 1.0376641750335693, -0.3735509216785431, -0.50816410779953, 0.48002663254737854, 0.10315433144569397, -0.3843402564525604, 0.019227806478738785, -0.12329627573490143, -0.9539615511894226, 0.08926554024219513, 1.8556275367736816, -1.440885066986084, 2.319390058517456, -1.6437044143676758, 0.08450501412153244, 0.19553406536579132, -1.1880086660385132, -2.952674150466919, -0.11093882471323013, 0.2516753375530243, -1.2717311382293701, -1.0329723358154297, 0.9865157008171082, 0.5311910510063171, 0.6295471787452698, 0.9163877964019775, -0.004995509050786495, 0.01128489337861538, -0.04824952781200409, 1.4549219608306885, -0.279701292514801, 0.09952564537525177, -0.170866921544075, 1.1292401552200317, -0.5774062275886536, -0.004988758824765682, -0.45566532015800476, -0.43337246775627136, 1.609336256980896, -2.2264328002929688, 0.08140871673822403, -0.041250668466091156, -0.29675930738449097, -0.43461596965789795, -2.1720075607299805, -0.00014107879542279989, -2.2554359436035156, -0.36712896823883057, -0.023113181814551353, 0.9328575134277344, -0.05219588056206703, 0.3879927098751068, -1.8086700439453125, 0.26316308975219727, -0.12510113418102264, -0.06180914491415024, 0.012854358181357384, 0.15846526622772217, 1.3681309223175049, 0.6640716791152954, 0.10186295956373215, 0.12698601186275482, -0.630171000957489, 1.8658397197723389, -0.9415164589881897, 0.2276535928249359, -0.09785082191228867, 1.082852840423584, 1.8424923419952393, 0.5638196468353271, -0.9493476748466492, -0.8808891773223877, -1.738256573677063, -1.4834322929382324, 0.49097880721092224, -0.6314650774002075, -0.8605936169624329, -0.004417391028255224, -0.3923262059688568, -0.012120394967496395, -0.5552725195884705, -1.707218885421753, 0.473321795463562, -1.4836407899856567, 1.674167513847351, 0.053854797035455704, 0.0008474017377011478, 0.06962870061397552, -0.11994145065546036, 1.6314098834991455, 0.3440045118331909, 2.646787166595459, -2.010892868041992, -0.08733956515789032, 0.8651012182235718, 0.9262235760688782, 3.0011115074157715, -0.20765772461891174, 0.5206525921821594, -0.563954770565033, 0.25785672664642334, 0.5228263139724731, -1.1570693254470825, -0.044976815581321716, 0.5005688071250916, -0.04951643571257591, -0.22922299802303314, -0.03901895135641098, 0.007726927287876606, -0.07901371270418167, 1.8088797330856323, 0.9734421968460083, 0.12000498920679092, 0.7241960167884827, 0.4358859956264496, -1.2748616933822632, 0.08504918962717056, 0.1495818793773651, -0.6870632171630859, -0.4933714270591736, 0.074973464012146, 1.3057247400283813, 0.8978766202926636, -0.11721202731132507, 0.5385451316833496, -0.49444320797920227, -0.01755562052130699, -0.576871395111084, -0.05941694974899292, -0.7291840314865112, -0.3592067360877991, -0.015598894096910954, 0.35624492168426514, -0.2201157510280609, 0.09255924820899963, 0.5325433611869812, 0.2219485342502594, -0.1500736027956009, 0.6758659482002258, -0.3101540505886078, -1.4669697284698486, -1.8100783824920654, -0.1058088019490242, 0.31868404150009155, 4.474277973175049, -1.2576689720153809, -2.882565498352051, 1.478513240814209, 0.5506777167320251, -0.3897309899330139, -1.2744200229644775, -0.10556884855031967, 2.462925910949707, 0.03658675029873848, -1.9555901288986206, 0.7115635275840759, -0.29812633991241455, 0.3763364553451538, -0.2144973874092102, -0.39680004119873047, 0.7336228489875793, 1.549423336982727, -0.19678664207458496, -0.02575141377747059, 0.7209508419036865, 0.015347322449088097, -1.7080920934677124, 0.038952380418777466, -0.08050260692834854, -1.254033088684082, 0.5992915630340576, 0.8213305473327637, -0.4443387985229492, -1.1298178434371948, -0.00044435038580559194, -0.8135229349136353, -0.4208775460720062, -0.7895951867103577, 0.3339376449584961, 0.13679540157318115, 2.3671140670776367, -0.01409454271197319, -0.03859425336122513, 0.21181198954582214, -1.7886110544204712, -0.08482073247432709, 0.33803725242614746, -0.22408835589885712, 0.1699671745300293, -0.5260392427444458, -0.1451522558927536, 0.29246464371681213, 1.723907232284546, -0.0700322762131691, 0.03800245374441147, 0.39268019795417786, -0.8089631199836731, 0.8105060458183289, -0.800939679145813, 0.41041457653045654, -0.6515111923217773, -0.321571946144104, -1.197588324546814, -0.08990183472633362, -0.09649623930454254, -1.184946894645691, 0.10219714045524597, 0.37086552381515503, -0.026412513107061386, 0.9078397154808044, 0.5120741724967957, -0.8351528644561768, -0.010020342655479908, 0.1615106612443924, 2.0938539505004883, -1.0557453632354736, -0.5367468595504761, -0.3797915279865265, -0.42144665122032166, 0.06008089333772659, -1.800668478012085, -0.3388127386569977, 0.27203160524368286, 1.1544630527496338, 0.8928064703941345, -1.4284701347351074, -2.3626596927642822, 0.07924024015665054, 0.022216007113456726, -0.31055861711502075, 0.05623101443052292, -0.22523900866508484, 1.3307698965072632, -0.890755832195282, -0.4558632969856262, -0.1532740592956543, -0.2895914316177368, -0.6774704456329346, -0.080734483897686, -0.12850208580493927, 1.1995059251785278, -1.558247685432434, 0.20106564462184906, -0.9588150978088379, -0.9048948287963867, 2.3232579231262207, 0.33374279737472534, 1.3092896938323975, -1.2353194952011108, -0.33031362295150757, 0.45009180903434753, -0.26716148853302, 1.950305461883545, 1.4427913427352905, 0.34361112117767334, -0.2447178214788437, -2.5016002655029297, 0.23549076914787292, -0.364082932472229, 0.346049427986145, -1.4617574214935303, -0.8361249566078186, -1.4239927530288696, 0.9107136726379395, -0.27125084400177, 0.768828272819519, -0.053262703120708466, 0.754319429397583, 0.051432155072689056, 0.15141978859901428, 3.0784246921539307, -0.04186423122882843, -0.04690854996442795, -0.003999981097877026, 0.003915038425475359, 0.9312766194343567, 0.49786752462387085, -0.43919268250465393, 0.4993845522403717, 0.43182554841041565, -0.5423659682273865, 0.018774792551994324, 0.16912996768951416, -0.5051267743110657, -2.9041428565979004, -0.056257255375385284, -0.12158084660768509, 0.003151162061840296, 1.6840474605560303, -0.35698753595352173, -0.7078235745429993, -0.06807860732078552, -0.5131205916404724, -0.14531151950359344, -1.1396307945251465, 0.9100664854049683, -0.21255840361118317, -0.09888540208339691, 0.42755240201950073, 0.027737397700548172, -1.686871886253357, -0.262834370136261, -3.854893684387207, -0.6963590383529663, -0.012700844556093216, 0.02837180159986019, -1.0719459056854248, -1.3520981073379517, -0.05551809445023537, 0.18127654492855072, -0.2756245732307434, 0.44052064418792725, 0.17118674516677856, 0.05089237913489342, 0.972061038017273, -0.002525567077100277, 2.4909448623657227, -1.3139915466308594, 0.15501768887043, 0.6094962358474731, 0.355843186378479, 0.17653746902942657, 1.3478199243545532, -0.8755864500999451, -0.7307108640670776, -2.3270950317382812, 1.4913452863693237, -2.400231122970581, 0.32688090205192566, 3.7105283737182617, 1.1685900688171387, -0.0867755264043808, -0.07837289571762085, -0.03332629054784775, -1.610527753829956, 0.7516413331031799, 0.031424812972545624, 0.17307312786579132, 0.07987675815820694, 0.33742842078208923, -0.21379011869430542, 0.11564038693904877, 0.34260159730911255, 0.1128607764840126, 0.18833789229393005, 0.2507947087287903, 0.27697935700416565, -0.029016034677624702, -0.950596809387207, -0.4948260486125946, 0.31595951318740845, -0.10660574585199356, -0.28668835759162903, -0.6266006231307983, 0.8215034008026123, -0.2401888072490692, -0.42696669697761536, -0.2754407525062561, 0.03264196217060089, -0.06874442845582962, 0.658465564250946, 0.019995780661702156, -0.13088864088058472, -0.6984548568725586, 0.01738925650715828, -0.6351441144943237, 1.2282100915908813, -1.3709161281585693, 0.363858699798584, -0.3159184157848358, 0.05959433317184448, 0.019380563870072365, 0.37018945813179016, 0.7080016136169434, -0.2823597490787506, 3.207648277282715, 0.14839786291122437, -0.1388121098279953, 0.20282599329948425, -0.05296269431710243, 0.5417993664741516, -1.5839531421661377, 0.40855690836906433, 1.0479812622070312, 0.13411875069141388, -0.6607879996299744, 2.225750207901001, 1.2562839984893799, -0.059208039194345474, 0.8095568418502808, 0.5892066955566406, -0.19352509081363678, 0.08333675563335419, 0.2642688453197479, 1.1794757843017578, -0.26123517751693726, 0.7275331020355225, -0.6989068388938904, 0.39259031414985657, 1.2969470024108887, -0.25868484377861023, -0.024839971214532852, -2.072509527206421, 0.4301469326019287, -0.257894366979599, -0.040348876267671585, -3.172421455383301, 0.6071125864982605, 0.5853715538978577, -1.296285629272461, 2.879600763320923, 0.011487198062241077, 0.17923177778720856, 0.1154763400554657, -0.24406182765960693, 0.12112145870923996, -0.04172346368432045, -1.2820073366165161, 1.0979571342468262, 0.381600022315979, -0.2331787496805191, 0.8527147769927979, -0.6702176928520203, 0.0031556147150695324, -1.1083515882492065, 0.07460460066795349, 0.15094192326068878, -1.223728060722351, 0.38267964124679565, 0.27582287788391113, 0.2286447137594223, -0.756267249584198, 0.12025430798530579, 0.6000230312347412, 1.8439979553222656, -0.2323422133922577, -0.5539577603340149, -0.025517582893371582, -1.6592248678207397, 0.704237699508667, -0.2763848304748535, -0.8480994701385498, -1.0478957891464233, -0.0476229228079319, -3.95121693611145, -0.5425306558609009, 0.40805700421333313, -0.15250413119792938, 1.963637351989746, -0.0978010892868042, -0.6409428715705872, -0.006966371089220047, 0.003093918552622199, -0.12169308960437775, 1.5734097957611084, 1.759953260421753, 1.4418823719024658, 0.4000284671783447, -0.2126120775938034, 2.1297383308410645, 0.38338086009025574, -0.1661628931760788, 0.2147214412689209, -0.21518833935260773, 0.025310484692454338, -0.1049840971827507, 0.0720248594880104, -1.1564158201217651, 0.7035946846008301, 1.3766412734985352, -0.10860289633274078, -0.671014130115509, 0.009141234681010246, -0.5053123235702515, 1.0600281953811646, -0.9817793369293213, -0.18741704523563385, 0.4682481586933136, -1.0735548734664917, -0.4685576558113098, 0.029363781213760376, -0.13518117368221283, -0.6421529650688171, -1.0939193964004517, 0.1130257397890091, 2.8132717609405518, -0.07359423488378525, -1.450830340385437, -1.3608213663101196, -0.09942934662103653, 0.13803620636463165, 0.1656101644039154, 0.12713709473609924, 0.8404157161712646, 0.004504145588725805, -0.024742312729358673, -0.20391349494457245, 3.228217363357544, -0.7379806041717529, -0.02854897454380989, -0.21337196230888367, 0.09998422861099243, -1.8071305751800537, -0.004471384454518557, 0.5889424681663513, -2.172025442123413, -2.3648762702941895, 1.504276990890503, -0.02069046162068844, 0.048396047204732895, -0.44857221841812134, 1.1242223978042603, 0.03753621131181717, 0.4439067840576172, 0.36148855090141296, 0.06689494848251343, -0.00468013621866703, 0.0062620993703603745, -0.6310838460922241, 0.3521310091018677, -0.7907993197441101, 0.1814095824956894, -3.0336170196533203, 0.009729303419589996, -0.005710771773010492, -0.1732511818408966, -1.0357342958450317, 0.3131650984287262, 0.8834095597267151, 1.1182832717895508, -0.42436668276786804, 1.3502380847930908, 0.9836606979370117, 0.12709838151931763, -0.3051144778728485, 2.1954169273376465, 0.017320074141025543, 0.2702733874320984, 0.037595368921756744, 0.07362230867147446, 0.17486990988254547, 1.3593943119049072, 2.027513027191162, 1.3421964645385742, 2.670832395553589, -0.9081528782844543, -0.2998717427253723, 0.100149966776371, -0.14923803508281708, -0.029984628781676292, 0.20321698486804962, -0.49864277243614197, -1.2170336246490479, 1.859660029411316, -3.2209079265594482, -1.55462646484375, 0.9105492234230042, -0.02859615534543991, -2.4316797256469727, 0.46120402216911316, 0.18125863373279572, -0.07599550485610962, 0.47840362787246704, -0.9351763725280762, -0.1435985118150711, 0.6449431777000427, -0.5426965951919556, -0.0341215580701828, 0.5037055015563965, 0.15610013902187347, -0.2733440399169922, -0.5743934512138367, 0.2958681285381317, 0.5075541138648987, -0.04859130084514618, -0.40354296565055847, 0.7193358540534973, 0.8033475279808044, 0.6593350768089294, -0.026613377034664154, -0.36300691962242126, 0.10725262016057968, 0.34847909212112427, -3.189189910888672, 2.1934256553649902, 0.287017822265625, -1.00816011428833, 1.197601318359375, 0.2455209493637085, 2.4775075912475586, 1.4987949132919312, -0.027837388217449188, -0.6787029504776001, 0.10396915674209595, 1.7963299751281738, -0.022761927917599678, -1.0714592933654785, 0.005412192549556494, -0.08551963418722153, -0.38603153824806213, -0.12477198243141174, 0.040629804134368896, 1.4820618629455566, -2.1764109134674072, 0.8199729323387146, 1.6812620162963867, 0.3765057623386383, 0.1586308777332306, 0.10118046402931213, 0.05001385137438774, 0.021636534482240677, 1.3078463077545166, -0.2771046757698059, -1.0452035665512085, 0.3979606628417969, 0.46242043375968933, -0.04890349507331848, -0.38312530517578125, 0.08361435681581497, -0.9913771152496338, 0.3776967525482178, -0.006415828596800566, -0.06885409355163574, 0.5044565200805664, -0.15573282539844513, -0.5249606370925903, 1.3459080457687378, -0.9143183827400208, -0.04770451411604881, -0.782820463180542, -0.3799491822719574, -0.005097634624689817, -0.29443398118019104, 1.246720790863037, -1.8189207315444946, 0.04714150354266167, -0.9652884602546692, 0.1093541756272316, -0.5744488835334778, -2.0146424770355225, -1.1371824741363525, 0.009110388346016407, -0.036225367337465286, 2.134875774383545, -0.0973600223660469, 0.9295227527618408, -0.0420524887740612, -0.964643120765686, 0.47759366035461426, 0.08505363762378693, -0.5248246192932129, -0.36029234528541565, 0.4696633517742157, -0.005861307960003614, 0.1795920878648758, 1.858250379562378, -0.21156425774097443, 0.05129455029964447, 0.054044708609580994, 2.0824272632598877, -0.7374500632286072, -0.5193390250205994, 0.016401255503296852, 2.3748085498809814, -0.6176171898841858, 0.09303687512874603, -1.2912331819534302, 0.07798294723033905, -0.6851437091827393, 0.9647425413131714, -1.698439598083496, 0.7408033013343811, 0.6335651874542236, 0.3854939043521881, -0.6529927849769592, -0.0005550926434807479, -4.754920482635498, 0.08749415725469589, -0.14418883621692657, 0.34016573429107666, 0.7782724499702454, -0.4612486660480499, 0.7367621660232544, 0.266557902097702, 0.560874879360199, 0.28453677892684937, 3.7455649375915527, 0.7802509069442749, 0.18398842215538025, 0.0041564032435417175, 0.03259663283824921, -1.610597014427185, -0.3883482813835144, 2.4135987758636475, 0.30828657746315, -0.3305051624774933, 0.012971031479537487, -0.7286139726638794, 0.4399626851081848, 2.090944290161133, -0.7333933711051941, -0.8921535611152649, -0.26998504996299744, -0.16657088696956635, 0.0015020336722955108, 0.11480523645877838, -3.4666242599487305, 0.3601396083831787, -0.821352481842041, 0.10791219025850296, 0.5302559733390808, 0.031308986246585846, -0.09491197019815445, 0.010888420045375824, 0.2320428043603897, -2.1577646732330322, 0.5990952849388123, 0.2399750053882599, -0.17210298776626587, 0.000520470377523452, 0.4375337064266205, -0.001345134573057294, 2.7305898666381836, 0.09172886610031128, -2.0904359817504883, 0.6742504239082336, 0.18995460867881775, 0.569564938545227, 0.039337627589702606, -0.8378255367279053, 0.11702799052000046, -1.495664358139038, 0.11629904061555862, 0.7469905614852905, -0.006040108855813742, 0.89320969581604, -0.14595764875411987, 1.1650538444519043, -0.6629601120948792, -0.3123670816421509, 0.2846287786960602, -0.6697810888290405, -0.45483362674713135, 0.5072880387306213, -0.8269187211990356, -0.9111704230308533, -0.5956472754478455, -0.006049135699868202, 0.8917626738548279, -0.01734287664294243, 0.926730751991272, -0.015032286755740643, 0.6191313862800598, 0.353850781917572, -0.4971473813056946, 0.15619097650051117, -0.9151880741119385, 0.59429931640625, 0.01994650810956955, -0.18075042963027954, -0.6291049122810364, -1.0522862672805786, 0.44738948345184326, 0.020578263327479362, 0.17796690762043, -0.04228326305747032, -0.4962373673915863, -1.0329052209854126, 0.8232782483100891, 1.057823657989502, 0.8443904519081116, -2.138775110244751, 2.294811964035034, 0.2630830407142639, -0.59884113073349, -0.12902551889419556, 0.9848052859306335, -1.3464487791061401, -0.22348704934120178, -0.0009010148933157325, 2.3810691833496094, -0.3026740550994873, -0.025056805461645126, -1.194204330444336, 0.11903921514749527, -0.5329001545906067, -0.8864679932594299, 0.007534494623541832, 0.03764917701482773, 0.1380687952041626, 0.9545613527297974, 0.09969969838857651, -0.09489037841558456, -0.1435638666152954, 0.797382116317749, -0.6879846453666687, 1.3202928304672241, 0.016370370984077454, -0.07531089335680008, -2.115983724594116, 0.07422833144664764, 0.29143989086151123, -0.041979942470788956, -0.5240828394889832, -1.3266123533248901, -0.028135260567069054, 0.36873525381088257, 0.21709632873535156, 0.34811151027679443, 0.44527125358581543, 0.0635533258318901, -1.4679487943649292, 0.00992813240736723, 0.0009155299048870802, 0.03374335542321205, -0.26613304018974304, -0.6421456336975098, 0.03109949454665184, 0.3318764269351959, -0.5922311544418335, -0.7145740985870361, -0.013975901529192924, -0.021044427528977394, -0.1123102530837059, -0.6985511183738708, -3.5165700912475586, -0.46262773871421814, -0.8962525725364685, -0.017786476761102676, -3.6424877643585205, 1.678276777267456, -0.2252357006072998, -1.1351357698440552, -1.2253278493881226, 0.4836949408054352, -0.1320764124393463, 2.657628297805786, 0.06031015142798424, -0.342844158411026, 1.6780444383621216, -1.072269320487976, -0.3537485897541046, 0.9160082936286926, 1.4689662456512451, 0.9035910367965698, -1.2342653274536133, -0.6734972596168518, 0.3002467751502991, 0.008543366566300392, 0.26401612162590027, -0.6273659467697144, -0.09359745681285858, 0.059997450560331345, 0.20712293684482574, -0.0603812001645565, 0.466473251581192, 2.740126609802246, -0.7813671231269836, -0.0031776900868862867, 2.8228912353515625, -0.3636593818664551, 0.45082584023475647, -2.6329421997070312, 1.0371463298797607, 0.019222917035222054, -0.33264854550361633, 2.179373264312744, -0.34736964106559753, -0.5002081394195557, -0.6483094096183777, 1.3652522563934326, -0.020490458235144615, 1.1238683462142944, -0.1120954304933548, 1.5636566877365112, 0.5957880616188049, -0.7947489023208618, 0.9869229793548584, -0.7424789071083069, 1.8365637063980103, 0.1715826392173767, -4.336605548858643, 1.2697864770889282, -0.5925689339637756, -0.7079417705535889, 0.28250911831855774, -1.853370189666748, 0.014982115477323532, 0.040727343410253525, 0.08730915933847427, 0.09314361214637756, -0.12199767678976059, 1.0985546112060547, 0.03732328489422798, 0.03549047186970711, 1.4842519760131836, 0.44296640157699585, -0.6269477605819702, 0.18804235756397247, -0.509977400302887, -0.004368988797068596, -0.806088924407959, 0.36576855182647705, 0.12832404673099518, -0.40265411138534546, -1.39281165599823, -1.6769201755523682, 0.3641858398914337, 2.38114070892334, -3.2521462440490723, -0.39389026165008545, 0.1081337183713913, 2.124101161956787, -0.18099616467952728, 1.3482879400253296, 1.7681645154953003, 1.1549468040466309, 1.292697787284851, 0.7826207280158997, -0.7150707840919495, 0.9290741086006165, -0.010704522021114826, 0.17252664268016815, 0.1138782799243927, 1.5366228818893433, -0.16735908389091492, -2.6047027111053467, 0.028800252825021744, 0.17450906336307526, 0.0025492378044873476, 0.008935101330280304, 3.300562620162964, 2.008803606033325, -1.5728306770324707, -0.8806127309799194, 0.1882443130016327, -2.310542583465576, 0.018155792728066444, -2.0385220050811768, 1.1789264678955078, -0.4827285408973694, -0.44318243861198425, -0.04892587289214134, 3.60961651802063, 0.0052094836719334126, 0.3195338547229767, -0.06131185591220856, 0.011637077666819096, 0.154657244682312, 0.6632689237594604, -0.16366046667099, -0.8160135746002197, -0.24343609809875488, 1.0986419916152954, 1.0410573482513428, -0.4827156066894531, -0.8828629851341248, 0.40590742230415344, -0.9518377780914307, 0.34557077288627625, 0.12412785738706589, -0.014811129309237003, -0.36423826217651367, 0.029384151101112366, -3.0468156337738037, -6.713805675506592, 0.32386502623558044, 0.03917350247502327, 0.45669886469841003, -0.8840001225471497, 0.21824948489665985, 0.04995107278227806, -0.4783189296722412, 0.06958990544080734, 0.1608385145664215, -1.6159297227859497, -0.8289018869400024, -0.2575785219669342, -0.4726333022117615, 0.13878725469112396, -0.017098406329751015, 1.020910620689392, -0.7040376663208008, 0.6057532429695129, -0.03072093427181244, 0.27742353081703186, 0.15118321776390076, -0.15154924988746643, -0.16941428184509277, 0.2263268530368805, 0.6280940771102905, 0.25698572397232056, -0.8906967639923096, 0.12288576364517212, -0.6386609077453613, -0.8935888409614563, 0.18205273151397705, -0.1058286651968956, -0.0029180010315030813, -1.0407564640045166, 0.4053407609462738, 0.16025100648403168, 0.21551302075386047, 0.45141685009002686, -0.41273659467697144, 0.187127485871315, 1.0362892150878906, -0.21212024986743927, 1.7055970430374146, 1.4462535381317139, -0.7168458700180054, -0.1954011619091034, -1.1344319581985474, 0.31098872423171997, 0.8735470175743103, 0.2982654571533203, -0.5183099508285522, -0.020820392295718193, 0.31204280257225037, 0.3440212309360504, -0.376869261264801, -0.06847264617681503, -0.17286086082458496, 0.29777446389198303, -0.44282400608062744, -0.9499979615211487, 1.2621606588363647, 0.062252286821603775, 0.42159420251846313, -0.9220223426818848, -0.31778576970100403, -0.0583035983145237, -1.3620808124542236, 0.4665380120277405, 0.0397685170173645, 5.078972339630127, -0.5022227764129639, 0.2179635912179947, 1.0462297201156616, -0.5317415595054626, -1.8275117874145508, -0.86253821849823, 1.3396260738372803, 0.07593637704849243, 0.22438094019889832, 0.28679895401000977, 0.1460389792919159, -0.22921225428581238, 0.5882654786109924, -1.1944818496704102, -0.36879459023475647, 0.20522445440292358, 0.08307766914367676, -0.707459568977356, 3.321784257888794, 0.016125284135341644, 0.028873465955257416, 0.21946865320205688, -0.07290913164615631, -1.695841908454895, -0.078086718916893, 0.9691863059997559, -0.04520651325583458, -0.2766100764274597, -0.22867217659950256, 0.9723561406135559, -0.000563593115657568, -0.14139802753925323, 0.519357442855835, -0.08602248132228851, -1.1422340869903564, -1.706924557685852, 1.4614591598510742, -0.41308262944221497, 0.3226667046546936, 0.6190927624702454, -0.3019104301929474, -0.3758935034275055, 3.0673038959503174, 0.4439912438392639, -1.2957191467285156, -0.45310500264167786, 0.03162666782736778, -0.01149742305278778, 0.7851770520210266, -1.6874068975448608, 0.07345043122768402, -0.016176417469978333, -0.043448805809020996, -0.39756667613983154, -0.031124042347073555, 0.38090503215789795, -1.6962883472442627, 0.3248341679573059, -0.261185884475708, -0.2802218496799469, 0.08773712813854218, 0.020452966913580894, -0.8567214012145996, -0.7302571535110474, -0.06168102100491524, 1.0334711074829102, -0.26552537083625793, -0.0646384209394455, -0.057577408850193024, 3.1285147666931152, 0.34631937742233276, 0.6953299641609192, 0.4646773934364319, 0.2157600224018097, -0.011690913699567318, 0.22668492794036865, 0.35535749793052673, -0.4537186026573181, -0.6577267050743103, 0.7778609991073608, 0.308877170085907, -2.5274763107299805, -3.869450807571411, 0.6051110625267029, -0.9736780524253845, -0.06515034288167953, -0.19448219239711761, 0.12576989829540253, -1.2867716550827026, 0.7273220419883728, 0.6505730152130127, -0.6370007991790771, -0.0455143079161644, 0.01711583323776722, 0.42538079619407654, -1.443554162979126, 0.08602054417133331, -0.12594743072986603, -0.18692289292812347, -0.5360482931137085, 0.3097304403781891, 1.0025371313095093, 0.9200769662857056, 1.078863501548767, -0.10226940363645554, 0.023775221779942513, -0.6013317704200745, 0.043116576969623566, 0.0952739417552948, -1.5899903774261475, -0.01718144491314888, 0.00808769091963768, 0.65125972032547, -0.08224330097436905, -0.018323998898267746, -0.1310824453830719, 1.0253708362579346, -0.34686926007270813, 0.09268195927143097, -0.06601002812385559, -0.2149326354265213, 0.844423770904541, -0.015859253704547882, 1.3615275621414185, 1.851311206817627, 0.2898864150047302, 0.26759257912635803, 0.326677143573761, -0.21671633422374725, 1.170667052268982, -3.156888961791992, -0.004206136334687471, 3.72226619720459, -1.2488020658493042, 0.11960426717996597, 0.3322724401950836, -1.133711814880371, 1.1542840003967285, -0.2993878424167633, 0.8227409720420837, -0.14534159004688263, 0.04769952595233917, 0.267297625541687, -0.8462509512901306, 0.7546717524528503, -1.2900627851486206, -0.20408691465854645, -1.2679271697998047, 0.11996947973966599, -1.1007381677627563, 0.37988755106925964, -0.5055461525917053, 0.38700056076049805, -0.8312470316886902, -0.1537112593650818, 1.6540237665176392, 0.7818698287010193, 0.9594149589538574, 1.554442286491394, -0.47600725293159485, 0.272921085357666, -0.1352057307958603, -1.1097540855407715, -0.8763232231140137, -1.1006269454956055, -1.6304467916488647, -0.7879267334938049, 0.6394243240356445, -0.4601312577724457, 0.19977137446403503, -0.6453346014022827, -0.05042123422026634, -1.4042255878448486, -0.1595984697341919, 0.3131835460662842, 0.7101153135299683, 0.7318164706230164, 0.24771510064601898, 0.8398218154907227, -2.1667871475219727, -0.02088714763522148, 0.2510143220424652, 1.4264655113220215, -0.000432997738244012, -0.22328953444957733, 0.00592928659170866, -0.07688190788030624, -0.4171944260597229, 0.34285488724708557, -0.0036599969025701284, -0.006761322729289532, 2.7678749561309814, -0.192745178937912, 0.20670874416828156, -0.09062493592500687, -0.04761165753006935, -0.3585827052593231, -1.2245758771896362, -0.12265308201313019, -0.5255823135375977, -0.006740848533809185, 0.2465958297252655, -0.7577660083770752, 0.6284980177879333, -0.18768662214279175, -0.6059286594390869, 3.5236971378326416, 0.8272271752357483, 1.1201542615890503, -0.4527601897716522, 0.4047524631023407, -0.5808938145637512, -0.11253373324871063, -0.610874593257904, 2.642129421234131, -0.6298058032989502, 0.5707218647003174, -1.2820442914962769, -1.0354185104370117, 0.0032091974280774593, -0.9792912006378174, -0.09502691775560379, 0.027985844761133194, 5.081989288330078, -0.7252376079559326, 0.3039635419845581, -0.08073526620864868, 0.12387792766094208, -0.11326905339956284, 0.23750998079776764, 1.719200849533081, -0.10890467464923859, -1.362895131111145, -0.41973307728767395, -0.026540610939264297, -1.2345731258392334, -0.23565050959587097, 1.3079133033752441, -0.7141607999801636, 0.2503206431865692, -0.2928304076194763, -0.31443700194358826, 0.47989168763160706, -0.6246320605278015, 0.0051977792754769325, -0.18207266926765442, -0.37492460012435913, -1.2800464630126953, -0.15642262995243073, 1.8929250240325928, -1.11654531955719, 1.448075771331787, -0.09638058394193649, 0.9106177091598511, -0.04845957085490227, -0.09327704459428787, 0.6748623847961426, 0.22231458127498627, 0.16781605780124664, 0.6936908960342407, -0.1821250021457672, 0.09925200790166855, 0.17471951246261597, -0.8766288161277771, -2.7085602283477783, -0.20876996219158173, -1.392180323600769, -1.427184820175171, 0.045651327818632126, -0.06964345276355743, -0.02620726265013218, -3.2976300716400146, 2.2220802307128906, -0.04946865141391754, -0.3596189022064209, -2.47235369682312, 0.45225515961647034, -0.8859566450119019, 0.004016818478703499, -1.3992689847946167, 1.6652672290802002, 1.0236622095108032, -1.1848845481872559, -1.184782862663269, 0.08024843037128448, 0.014125190675258636, 0.05576755478978157, 0.02807968109846115, 0.5821050405502319, 1.3086190223693848, 0.42764532566070557, -0.4897410273551941, 0.6480957865715027, -0.6401326060295105, -1.1777969598770142, -1.053816556930542, -0.028033411130309105, -3.9403092861175537, 0.565750777721405, 0.13002637028694153, -3.312715530395508, 0.16587966680526733, 0.08270107209682465, 0.2761595547199249, 1.1376326084136963, 1.515477180480957, -0.045127931982278824, 0.7036440968513489, 1.9365848302841187, 1.1119236946105957, -0.5314009785652161, -0.07029502093791962, -1.986646294593811, 2.0016000270843506, -2.525524616241455, -0.5574681758880615, 0.4709438979625702, -0.37227627635002136, 0.08830641955137253, 0.8871225118637085, 0.1711539328098297, 0.5294912457466125, 0.2892812490463257, 0.00547182559967041, 0.296907514333725, -1.163926362991333, -0.2991916239261627, -0.1412220299243927, 0.2926918566226959, -0.005521750543266535, 1.1555004119873047, 1.236690878868103, 0.5797953605651855, 0.260518878698349, -0.828184962272644, 0.15116406977176666, -1.1469181776046753, -0.03016091138124466, 0.14383064210414886, -0.2391132116317749, 1.1814236640930176, 0.18606697022914886, -0.12193392217159271, 0.3597683906555176, 0.04639994353055954, -0.01942448690533638, -0.5579138398170471, 0.3120988607406616, -0.46908843517303467, 2.0927836894989014, -0.5368279218673706, -1.0101139545440674, -0.14590507745742798, 0.24979327619075775, 0.3267689347267151, 0.3635065257549286, 1.5440576076507568, 1.0486469268798828, -0.020153595134615898, 0.011217191815376282, -0.11660565435886383, -0.874394953250885, -1.5489330291748047, 0.3778860867023468, 0.007802276406437159, -0.18652953207492828, -1.064466953277588, 1.1799571514129639, 0.00310039846226573, -0.47764626145362854, 1.1471035480499268, -0.0409034863114357, -1.5936709642410278, 2.3868277072906494, -3.306001663208008, 0.1214304119348526, 1.0445516109466553, 0.11420764029026031, 1.6220667362213135, 0.10121461004018784, -0.5577580332756042, 0.10449174046516418, -1.1395601034164429, 0.7575612664222717, 0.03234993293881416, 0.8312609195709229, -0.08850423246622086, -0.678763210773468, 1.3461241722106934, 0.6435340642929077, 0.2199375182390213, 0.6400148868560791, 0.792407214641571, 1.8429025411605835, -1.1781517267227173, 0.20867058634757996, 0.21075625717639923, 1.1348401308059692, -3.2205209732055664, -1.9836046695709229, -0.3846922218799591, -0.22164639830589294, -0.4055234491825104, 0.7662295699119568, -0.2777634859085083, -2.733957052230835, -1.1939440965652466, 1.3097286224365234, -0.4608476459980011, -0.07982580363750458, -0.22792948782444, 1.0999817848205566, 0.17416267096996307, 0.05444036051630974, 0.04688451811671257, 1.1306991577148438, 0.5685451030731201, -0.921270489692688, -0.7014124393463135, 0.0941351056098938, -0.19157415628433228, -0.14059635996818542, -0.004015706945210695, 0.2923355996608734, 1.1942635774612427, -0.3003104031085968, 1.1656311750411987, -0.09204508364200592, 0.37123703956604004, -0.4617309868335724, 0.09674994647502899, 1.2128539085388184, 0.6406478881835938, -0.19019663333892822, -4.007089614868164, -0.021725520491600037, 0.46682870388031006, 0.17177292704582214, 0.017418092116713524, -0.31943872570991516, -0.11104843765497208, -1.0926637649536133, 0.9878093600273132, -3.2239112854003906, -0.5949321389198303, 1.2606812715530396, -0.25379130244255066, 0.4570125341415405, 0.8737452626228333, -0.12436424195766449, -0.24620448052883148, 0.6667227745056152, 0.1285122036933899, -1.136975884437561, 0.08299527317285538, 0.03262375667691231, 0.8943694233894348, 0.3779274523258209, -0.40988489985466003, -0.21803316473960876, 1.106257438659668, -0.21927602589130402, 0.4470968544483185, -0.94521164894104, 1.2230966091156006, -0.08086765557527542, -0.15544743835926056, 1.2209819555282593, 0.4002169072628021, 0.45261406898498535, -0.22417181730270386, 0.5614712238311768, -0.02888171747326851, 0.2022097110748291, -1.1174899339675903, 0.015147250145673752, -0.24960081279277802, 0.3274657726287842, 0.40851324796676636, -0.6715060472488403, 0.1488712728023529, 0.07171498239040375, 0.07796172052621841, -0.24432820081710815, -0.46000900864601135, 0.2658468782901764, -0.04194263368844986, -1.4394841194152832, 0.2154088318347931, 0.48092103004455566, -0.0060198623687028885, 0.6392146348953247, 0.10796017199754715, -0.42726948857307434, -0.09245534986257553, -0.01355128176510334, 0.1415388137102127, 0.9162450432777405, -0.6467413902282715, 0.8875010013580322, -2.0398762226104736, -0.03624507784843445, -0.270379900932312, 0.28888654708862305, -3.7519326210021973, 0.44857069849967957, -0.5433974266052246, 0.5825767517089844, -0.04832955822348595, 0.038071054965257645, -0.15097396075725555, -0.06516999751329422, -0.09319278597831726, -0.029441308230161667, 0.9439792037010193, -0.22513999044895172, 0.5832037925720215, -0.0009777903323993087, 0.6756088137626648, -0.4871746003627777, -1.3037774562835693, -0.42867788672447205, 0.11680252850055695, 0.06201304867863655, -0.6161158084869385, 0.9821538329124451, -0.012818850576877594], [-1.1503546237945557, -0.7399186491966248, -0.017890268936753273, -1.6515095233917236, -0.9499272108078003, -0.1882692575454712, 0.5478187203407288, 0.624086856842041, -0.008640171028673649, -0.14906355738639832, 0.12021520733833313, 0.958467960357666, 0.863633394241333, 2.4284398555755615, -1.7164015769958496, -0.22509631514549255, -1.38165283203125, 1.2328931093215942, 0.01866251602768898, -0.044891566038131714, -1.3024793863296509, 0.1483708769083023, 1.4358490705490112, 0.07358214259147644, 0.9009535908699036, 1.2760246992111206, 0.7265660762786865, -0.18183456361293793, 0.30910924077033997, 0.15067312121391296, -0.226154163479805, 0.6371912956237793, 0.9922523498535156, -0.01725860685110092, 0.13015130162239075, -0.3251383602619171, -0.4382423460483551, 0.5819075703620911, 0.03349137678742409, -1.8852084875106812, 1.0231329202651978, 0.011561625637114048, 2.020977020263672, 0.02725081332027912, -0.9700355529785156, -0.7564339637756348, 0.38362830877304077, 0.2059956192970276, -0.06453181803226471, 0.07915067672729492, 2.2122907638549805, -1.1503362655639648, 1.0667822360992432, -0.6553170680999756, -0.1320062279701233, -1.0954623222351074, 0.18943922221660614, 0.9423784613609314, 0.28033989667892456, 0.44674935936927795, 0.7461209297180176, 0.6286872029304504, 0.06237592175602913, 0.2618415653705597, 0.0819375142455101, 0.16520093381404877, -1.3867502212524414, -2.6054139137268066, -0.5630525946617126, -0.6390970945358276, -1.330628514289856, 2.006863832473755, -0.5980181694030762, 0.6891207695007324, 1.1125456094741821, -0.5780991315841675, 0.8840680122375488, 0.002812095684930682, -0.10473539680242538, -0.018921028822660446, -0.2473950982093811, -1.1281718015670776, 1.0880974531173706, 0.012639962136745453, 0.1499723494052887, -0.020961934700608253, -0.13663430511951447, 0.12635791301727295, -1.1254053115844727, 0.10277068614959717, 0.07429610192775726, -1.5080451965332031, 0.05127245932817459, 0.37849244475364685, -0.08776220679283142, -0.9235985279083252, -0.4382760226726532, 0.2607933580875397, 0.2691299617290497, 0.017675701528787613, 1.4440046548843384, -1.9228774309158325, 0.07410039007663727, 0.06481344252824783, 0.006861325819045305, -0.023734379559755325, 1.751246452331543, 1.3632129430770874, 0.00022053434804547578, -0.7017565965652466, -0.27280640602111816, -0.20364150404930115, -2.3346385955810547, -0.3775924742221832, -0.15338532626628876, 0.04641092196106911, 0.18895472586154938, 0.282604843378067, -0.10502326488494873, 0.11185085773468018, -1.8361141681671143, -0.02200239896774292, 0.027327677235007286, 0.18847402930259705, -2.1776909828186035, -0.13456185162067413, -0.57480388879776, 0.32549744844436646, -0.34446054697036743, -0.05175456777215004, -0.5670594573020935, 0.005014350172132254, -0.5462570786476135, 0.3300555944442749, 0.18252602219581604, -0.21528036892414093, -0.006611554883420467, -0.22776195406913757, -0.0332740880548954, 0.04780774936079979, 0.3410860002040863, 0.6907472014427185, 0.6454703211784363, 0.07342411577701569, 5.555062294006348, -0.06200025975704193, 0.719960629940033, -0.5008652806282043, 0.14179560542106628, -1.1753041744232178, -2.7865068912506104, 0.15473826229572296, -0.8911239504814148, 0.36645448207855225, 1.0835473537445068, -1.3580188751220703, 0.4661432206630707, -0.17170144617557526, 0.5588619709014893, -1.4382827281951904, 2.703234910964966, 0.14823280274868011, 0.10144257545471191, -2.4299540519714355, 0.1242685616016388, -1.4406800270080566, 0.2891339361667633, -0.06137635558843613, -0.6288290619850159, -0.5934414267539978, 1.2328789234161377, 0.11945609003305435, 2.4726369380950928, -0.3864557147026062, 1.986311435699463, -0.0023781917989253998, -0.4948958158493042, -0.4700371325016022, -0.14522765576839447, -1.6141188144683838, -0.1318989247083664, -0.006513537373393774, -0.5180470943450928, -0.4403091371059418, -0.37503644824028015, 0.2260802835226059, -0.003883709665387869, 0.0035983461420983076, 0.7253862023353577, -0.004606024362146854, -1.3999520540237427, -0.04362327232956886, -0.00019540064386092126, 1.2666319608688354, -0.04811364784836769, -0.04164665564894676, 0.26477038860321045, 0.5429180860519409, -0.09196475148200989, 0.03487979620695114, 0.03208165615797043, -0.018921181559562683, -0.05629691481590271, -0.34216082096099854, -0.38353678584098816, -0.057954177260398865, -3.662717580795288, -0.5229921340942383, -1.4379451274871826, -0.5813194513320923, 0.2917625904083252, -1.1635239124298096, 0.43696221709251404, -0.30308040976524353, -0.381607323884964, 0.26957571506500244, -4.320422172546387, 0.1470649540424347, 0.630777895450592, 0.35527727007865906, -0.6515634655952454, 0.10067079961299896, 0.011654727160930634, -0.7149448394775391, 0.5484649538993835, 0.49620625376701355, -0.929709255695343, 2.0560688972473145, 1.766535758972168, 0.1357058733701706, 0.009961449541151524, -0.16996198892593384, 0.46843791007995605, -0.7543647289276123, 0.2665072977542877, -1.1605740785598755, 0.11601081490516663, -0.4783729314804077, -0.3706256151199341, 0.12049591541290283, -1.651670217514038, 0.24530619382858276, 0.6744602918624878, 0.1995580941438675, 0.021471019834280014, -0.09597927331924438, -0.5119701027870178, 0.14152264595031738, -0.15975604951381683, 0.5991950631141663, 0.1791038066148758, -1.260396957397461, 0.7332550883293152, -0.023443913087248802, -1.9717648029327393, -0.1917475163936615, 0.12529049813747406, 0.33925968408584595, 0.014216899871826172, 0.4592822194099426, 0.1435365527868271, -0.053820718079805374, 1.8855384588241577, -0.14477761089801788, 0.5579645037651062, -0.22789394855499268, 0.4337104558944702, 0.4680255949497223, -1.308090329170227, 0.8599564433097839, -0.5430183410644531, 0.09772618114948273, -0.4221055507659912, 0.13739937543869019, -0.0013896211748942733, 0.08722149580717087, -1.6492146253585815, 0.42053940892219543, 0.0837852880358696, -0.4360327124595642, 0.2497738003730774, 0.05534975975751877, -0.25293317437171936, 0.3118763864040375, -0.03663958981633186, -0.1454540640115738, 0.29183563590049744, -0.04156405106186867, -0.5016089081764221, 1.0901957750320435, -1.4299228191375732, 0.40799468755722046, -0.1531195342540741, -0.3231053948402405, -0.23066382110118866, -1.5146335363388062, -0.516814112663269, -3.8479909896850586, 0.8066173791885376, -0.41591283679008484, -0.48580634593963623, 1.098353624343872, 0.16796955466270447, 0.2145065814256668, -0.14671935141086578, -0.34298449754714966, -0.7463213801383972, -2.1329848766326904, -0.584646463394165, -0.8831837773323059, 1.122227430343628, 0.6219214200973511, -0.007232637610286474, 0.29747703671455383, -0.7924982905387878, 2.516352653503418, -0.295945942401886, -0.16838304698467255, -0.17509189248085022, -0.7740378975868225, -0.6272192597389221, -0.16914395987987518, 0.4455516040325165, 0.17701908946037292, -1.459169626235962, 0.8430718183517456, -1.1472584009170532, -0.4549120366573334, -0.06855147331953049, 0.12889324128627777, -0.601712167263031, -0.060056205838918686, -0.0023219508584588766, -1.1254786841163877e-05, -0.5802275538444519, -0.1759624183177948, 0.0030508749186992645, -0.07901781052350998, 0.054700929671525955, 0.8215312361717224, 0.44932469725608826, -1.8884178400039673, 1.275593638420105, -1.8679252862930298, 0.9774262309074402, 0.24341417849063873, -0.37461477518081665, 0.1045190840959549, 0.20405055582523346, 0.7617365121841431, -0.10439124703407288, 0.20227298140525818, 0.16232380270957947, 0.8208408951759338, 0.31556397676467896, -0.10237409174442291, 2.3542962074279785, 2.1842379570007324, 0.035037264227867126, -0.09718643873929977, 0.5298277735710144, 0.3401738703250885, 0.38261744379997253, 0.06818225234746933, -0.467662513256073, -0.29053643345832825, -0.25045666098594666, 0.07151836901903152, 0.277847021818161, 1.3277943134307861, 0.06871072202920914, 1.1114338636398315, 0.3512240946292877, -0.17240208387374878, -0.6875256896018982, 0.19950056076049805, 0.0878794714808464, 0.03829898312687874, -0.1301264464855194, 0.46141737699508667, 0.03141159564256668, -0.4390603005886078, -0.8336803913116455, -0.00889874342828989, 0.10346716642379761, 0.08141722530126572, 0.3025134205818176, -1.8846276361728087e-05, 0.6500462889671326, 0.0688575878739357, -0.4852786958217621, 0.4978935420513153, -0.4366796016693115, 1.0654834508895874, 0.6009646654129028, -0.8340700268745422, -1.8468093872070312, -2.133448362350464, 0.3689378798007965, -0.8170962333679199, 0.0909232571721077, 0.0017346394015476108, -0.13184817135334015, 0.6100212335586548, 0.03495008870959282, -0.8389233946800232, 0.24372009932994843, 0.27043160796165466, -1.098571538925171, -2.8212966918945312, -0.1539849042892456, -0.395797461271286, -0.0336795337498188, -1.075014591217041, 0.28197041153907776, 0.8619029521942139, 0.03020908497273922, 0.16342544555664062, -0.0975630134344101, -0.06194877624511719, 0.18569014966487885, 0.13421200215816498, -0.15384773910045624, 0.07328886538743973, -1.913018822669983, 0.005670212674885988, -1.1991149187088013, 1.7574055194854736, 0.261889785528183, -0.4312852919101715, -0.28546178340911865, -0.305928498506546, -0.3659118413925171, -0.049287572503089905, -2.546984910964966, -0.013193692080676556, 0.8271703124046326, -0.1195274218916893, -0.4083096981048584, -0.32582414150238037, -0.18028713762760162, 0.032422758638858795, 1.9781146049499512, -0.03604021668434143, -0.5660465955734253, 1.2027206420898438, 0.8446298837661743, -0.1495252549648285, 0.14646784961223602, 0.026310555636882782, 0.46414870023727417, -0.1292307823896408, 0.7369653582572937, 0.012157357297837734, 0.24277062714099884, -0.842887818813324, -0.1847904771566391, 1.243379831314087, 0.4059641659259796, 0.5435171127319336, 0.3159576654434204, 0.07424032688140869, 2.7544643878936768, 0.6623949408531189, 1.4582688808441162, -0.09392684698104858, 0.8527289032936096, 0.07589756697416306, -0.1800241321325302, 0.00013911824498791248, -0.1602175384759903, -0.6332539916038513, -0.45682060718536377, 0.007508912589401007, 0.08239218592643738, 0.4158872961997986, -0.3685925304889679, 0.637894332408905, 0.698022186756134, -0.0721820816397667, 1.7854300737380981, -1.0919352769851685, -0.8978126049041748, 0.48960867524147034, 0.2771202325820923, -0.08620838075876236, 0.18387499451637268, -0.10967887938022614, -0.39291873574256897, -1.2635692358016968, 0.26775306463241577, -0.21870185434818268, -0.07910778373479843, -1.1470485925674438, 0.8944279551506042, 0.13739383220672607, -2.0880823135375977, -0.08653967082500458, 2.031684637069702, -0.29906895756721497, 1.0474331378936768, 0.54732346534729, 0.5782259106636047, -0.318859726190567, -0.29958629608154297, 1.6217279434204102, 0.5707678198814392, 0.9033117294311523, -0.3002564311027527, 2.8408823013305664, 0.33814477920532227, -0.05679556727409363, -0.03198634460568428, -0.02123424783349037, -1.3117936849594116, -0.013661136850714684, -0.08590012788772583, 0.41380614042282104, -0.6296723484992981, 0.22725796699523926, -0.0025850459933280945, -1.650118112564087, -2.2628629207611084, 0.9463843703269958, -0.227175772190094, -0.39582890272140503, 0.016335556283593178, -0.294024258852005, -0.9376101493835449, 1.0552810430526733, -0.013576661236584187, 2.8631982803344727, -0.27676063776016235, 0.019801612943410873, 1.5776149034500122, -0.16484317183494568, -1.6176244020462036, 0.8499243855476379, 0.792721152305603, 0.08644010126590729, -0.04684457555413246, 0.021620407700538635, -0.7137373089790344, -1.1274040937423706, 0.03686682879924774, 0.03552311658859253, -0.0541456937789917, -0.9033193588256836, -0.41081511974334717, -3.8196613788604736, -0.4015555679798126, 0.1510884314775467, 1.0466978549957275, -0.2599789798259735, 0.11279001086950302, -0.6501619815826416, 0.19126830995082855, 0.684844970703125, 3.33646559715271, -0.150496244430542, -2.773287773132324, -1.3180488348007202, -0.29753661155700684, -0.7478297352790833, -0.004713159520179033, -0.03790280222892761, 0.7185757160186768, -0.905484676361084, 0.25415071845054626, -0.7208064794540405, 0.05937400087714195, 0.10363322496414185, 0.008230967447161674, -0.02429758943617344, -3.2358169555664062, 0.09823622554540634, -0.9069036841392517, 0.46200457215309143, -0.03617175668478012, 0.7455376982688904, 1.6096758842468262, 3.004058361053467, 0.5186336040496826, -0.36147797107696533, -0.4908412992954254, 0.3138091564178467, 1.032091498374939, -0.2988559305667877, -0.17697440087795258, 0.7561291456222534, 0.016216406598687172, -1.1125941276550293, -0.15365582704544067, -0.5622323155403137, -0.15193918347358704, 2.6165530681610107, -0.4841488301753998, 0.10754159092903137, 0.9485882520675659, 0.03410780057311058, -0.9417853951454163, -0.17677482962608337, -0.09401707351207733, 0.41467517614364624, 0.44993504881858826, 0.0029832629952579737, -1.1096457242965698, 0.780778706073761, 0.10284798592329025, 2.24027681350708, -0.1196785420179367, -0.9027987718582153, -0.7354140877723694, -0.07014212012290955, 0.3424602448940277, 0.07451220601797104, 0.09717200696468353, 0.24423646926879883, -2.2156107425689697, -0.2654823064804077, -0.3005193769931793, 0.33409175276756287, -1.662732720375061, -1.998965859413147, -0.8951613306999207, 0.9856014847755432, -0.4510645866394043, 1.5670998096466064, -0.20165620744228363, -0.17069052159786224, 0.30700743198394775, 1.2191779613494873, 0.3723197281360626, -0.5949872732162476, -0.38951921463012695, -2.9776523113250732, -0.14113757014274597, -1.0021439790725708, -0.29475829005241394, 0.8874574303627014, -1.0658917427062988, -0.01372522208839655, -1.9139608144760132, -1.2681294679641724, 0.21363729238510132, -0.9319267272949219, -1.5514897108078003, 0.31708982586860657, 0.06228318810462952, -0.5343913435935974, 0.0037137221079319715, -0.5824320912361145, 0.43777620792388916, -0.1422392874956131, -1.9068270921707153, 0.9562683701515198, 0.3455401062965393, -1.538973331451416, 0.9420398473739624, -0.660538375377655, 0.4068472385406494, -0.13410983979701996, -1.0054445266723633, -0.09549989551305771, -0.10270026326179504, 2.0647895336151123, 0.011026284657418728, -0.15033367276191711, -0.25714361667633057, -1.3572430610656738, 0.02395867556333542, -0.7342033982276917, 0.1953737586736679, -1.3016245365142822, 0.19660907983779907, 0.07059945166110992, -0.10292676836252213, -1.3902735710144043, 2.134263277053833, 3.261345148086548, 0.10184837877750397, -0.1565665453672409, -2.283099889755249, -0.7369014620780945, 0.015267704613506794, 0.9088494181632996, 0.0739775076508522, -0.15690778195858002, -0.0834420695900917, 2.0982813835144043, 0.6777202486991882, 0.00016237888485193253, 0.18119369447231293, -0.038993291556835175, 2.0147452354431152, -0.10257451236248016, -1.2874820232391357, -0.12408195436000824, -1.537591814994812, 1.0583723783493042, 1.3470094203948975, 0.8378386497497559, 0.012160352431237698, -0.15903589129447937, 0.547301709651947, -4.538564205169678, -0.27718836069107056, -0.027649108320474625, -0.47904327511787415, 0.7643864154815674, 2.153400421142578, -2.3423304557800293, 0.01325957290828228, 0.23602962493896484, 2.1265809535980225, -0.0066627864725887775, 0.25823691487312317, -2.027960777282715, 0.5481258630752563, 0.24571408331394196, 0.05297928303480148, 0.3602280616760254, 0.9975078105926514, 0.2454446256160736, -0.08109157532453537, -0.2526766061782837, -0.05887552350759506, -0.09487754851579666, 0.2436019480228424, 2.303694248199463, -1.092903971672058, -0.14773117005825043, 2.1529901027679443, -1.105278730392456, 0.13007844984531403, 0.10520529747009277, -0.37510114908218384, 1.1536402702331543, 0.6959801912307739, -0.40407490730285645, -0.0006503431359305978, 2.020111083984375, -0.7837442755699158, -1.0493232011795044, 0.8406530022621155, -0.2818402945995331, 0.7382505536079407, -0.4448801875114441, 0.5784566402435303, -1.5293134450912476, 1.2775441408157349, 1.0205656290054321, 0.5685813426971436, 1.0195659399032593, 0.04661252349615097, 0.7314379811286926, 0.31240999698638916, -0.3804944157600403, -0.011428321711719036, 0.0037456045392900705, -0.1446293741464615, 0.12234697490930557, -2.1579058170318604, 1.71930730342865, -0.21955186128616333, -0.005170061718672514, -0.03341495618224144, -0.03814072534441948, -0.7061396837234497, 0.6784616112709045, -0.0009765294380486012, -1.2659337520599365, 0.002243766328319907, -0.3247182369232178, 0.5343717336654663, 0.718961238861084, -0.035709187388420105, -0.18808291852474213, 0.48162487149238586, 2.7782578468322754, 0.04292572662234306, -0.26461556553840637, -0.5883317589759827, 0.5976719260215759, -0.14957062900066376, -1.2803566455841064, 0.9108558297157288, -1.0627022981643677, -1.0647857189178467, -0.026556462049484253, -0.3515450060367584, -0.6895961761474609, -0.4490795135498047, -0.015300652012228966, -0.15968865156173706, 0.43315720558166504, -1.2245110273361206, 0.24036604166030884, -0.307326078414917, -0.8267772197723389, -0.0010884007206186652, -0.7849034070968628, -0.7009798288345337, 0.1116371750831604, -0.10713652521371841, -1.3343511819839478, 0.5549231171607971, -0.6136960983276367, -0.8680447340011597, 0.49448153376579285, -0.6246336698532104, 0.4629383981227875, 0.5376818180084229, 0.4048081338405609, -0.6720741987228394, -0.1921883523464203, 0.39242634177207947, -0.2378387302160263, -0.01071565505117178, -1.687983751296997, 0.011923597194254398, -0.4683767259120941, -0.6535040140151978, 0.08397061377763748, -0.2842026352882385, -2.7657573223114014, -0.3797578811645508, -0.14366787672042847, 0.3217529356479645, -0.15952816605567932, 0.06985689699649811, -0.09266342967748642, 0.24662649631500244, -4.200842380523682, 0.5027207732200623, -0.44161656498908997, -0.04892336204648018, -0.7815912365913391, -1.2525380849838257, 0.2139672338962555, -0.004793361760675907, 0.07577311992645264, -0.5650612711906433, 0.15183795988559723, -0.5077812075614929, 0.6368260979652405, -0.10600300878286362, -0.015914427116513252, -0.09820222854614258, 0.07965723425149918, 0.35243263840675354, 0.6449657678604126, -0.5480422973632812, 0.3636743426322937, -0.23634657263755798, -0.040422894060611725, -1.0303354263305664, -0.09095169603824615, -1.7798843383789062, 0.0022570223081856966, -1.4240800142288208, -0.014148552902042866, -0.4460279643535614, -0.10376745462417603, -0.05989501625299454, -0.5865346193313599, 0.747120201587677, -0.38381144404411316, -1.561719536781311, -0.7955595254898071, -0.32097315788269043, 0.9930971264839172, -1.8752639293670654, 0.1682344377040863, 0.8766670227050781, 0.5971482992172241, -0.07257988303899765, -0.30665698647499084, 0.49321508407592773, -0.8766129016876221, 0.33530548214912415, 0.2417038530111313, -0.9885779023170471, 0.08843696862459183, -0.240151047706604, 0.42320024967193604, 0.06528379023075104, 1.0420935153961182, -0.7386671304702759, -0.472927063703537, 0.9128237962722778, -0.25112298130989075, -1.3866939544677734, 0.9475502967834473, 0.36002692580223083, 2.2375431060791016, -0.00500342296436429, -0.13130497932434082, -0.12094098329544067, 0.48250651359558105, -0.7371196150779724, -0.0677281841635704, 0.43569982051849365, 0.5926867127418518, 0.11007265746593475, 1.0642132759094238, 0.0924597829580307, 0.18147152662277222, -0.014844956807792187, 0.7121477127075195, 0.04121248424053192, -0.2910102605819702, 1.4215052127838135, 0.46454697847366333, -0.9799526333808899, 0.14277730882167816, -0.5343262553215027, -0.23270303010940552, -0.02767304703593254, -0.5814378261566162, -0.38936173915863037, 0.23317132890224457, -0.14183519780635834, 1.7000669240951538, 0.2041417360305786, -0.3595382273197174, 0.5276694893836975, 0.5137524604797363, 0.05556982383131981, 1.3928974866867065, 0.11409588158130646, -2.6304616928100586, -0.15941466391086578, -3.746171236038208, -0.388639897108078, 0.5477138757705688, -0.02219892293214798, 0.010230413638055325, 0.1563778519630432, -4.473182201385498, -0.3209178149700165, 0.2465181052684784, 0.0609518364071846, -0.4549773931503296, 0.7148404717445374, 0.17721699178218842, -0.10395363718271255, 0.34814953804016113, -0.39933282136917114, 0.04647252708673477, -0.3566075563430786, 0.7384331822395325, -1.2277421951293945, 1.155276894569397, 1.0154244899749756, -0.07598021626472473, 0.05785032734274864, -0.08179686218500137, -0.2158152163028717, 0.6110013723373413, -1.2613680362701416, 0.7253409624099731, 1.3098570108413696, -0.3635624051094055, -0.519623875617981, -0.012757726013660431, -0.475396066904068, -0.4843875765800476, 1.0332540273666382, -0.24127686023712158, -0.38394302129745483, -0.26743462681770325, 0.6132818460464478, 1.232706904411316, -0.04660391807556152, -1.4733222723007202, 0.07082806527614594, -0.8069868683815002, -0.5171929597854614, -0.8806251287460327, 0.004507276229560375, -0.7424198389053345, 0.4537300169467926, -0.37921446561813354, 0.4424702823162079, 0.39769965410232544, -0.17362600564956665, 0.08212858438491821, 0.02770315483212471, -0.3184826970100403, -1.5192718505859375, -0.5834939479827881, 1.0857906341552734, 1.1078670024871826, 0.058851879090070724, 1.7484400272369385, -0.3420713245868683, -1.5336993932724, -0.6190149784088135, -0.0020936124492436647, -0.046014558523893356, 0.038562994450330734, -0.1396617740392685, -0.7840375900268555, -0.7078657150268555, -0.2554519474506378, 0.6442837119102478, 3.271122932434082, -0.06445378065109253, -0.05326297506690025, 0.15448616445064545, 1.0807472467422485, -0.002726014470681548, 0.10164496302604675, 0.5665382742881775, -0.24579599499702454, -2.9269535541534424, 1.8811135292053223, -0.8782835602760315, -0.2569916248321533, 3.5927627086639404, 0.013902520760893822, 0.18203365802764893, -0.11938313394784927, 0.008553864434361458, -0.1035405620932579, 0.5833511352539062, -2.76396107673645, 0.5658439993858337, 0.7790796756744385, 0.5181930661201477, -0.2625744342803955, 0.02646728977560997, -0.505983293056488, 0.02725697122514248, -0.04107855632901192, 0.6696933507919312, -0.09939587116241455, 0.38179290294647217, 0.445320188999176, -2.095733880996704, -1.4593117237091064, 0.4644635021686554, 0.7757909893989563, -1.168871283531189, -0.256942480802536, -0.376892626285553, 0.47187501192092896, -0.05683014541864395, 0.026266371831297874, -0.9789149761199951, -1.378678321838379, 0.3420633375644684, -0.17233040928840637, -0.6703803539276123, 0.016586316749453545, -0.45912790298461914, -0.7038560509681702, -0.12824158370494843, 0.5232312679290771, 0.44303378462791443, -2.1823770999908447, 0.12182920426130295, -0.24844616651535034, 0.23384155333042145, -0.7886896729469299, 1.0710231065750122, 0.3780696988105774, 1.3691037893295288, -0.7822365164756775, -2.128110647201538, 0.3201901912689209, -0.13146670162677765, -0.39902302622795105, -0.6413205862045288, 1.9075323343276978, 0.9015538096427917, -0.043449368327856064, 0.009443279355764389, 2.2955164909362793, -0.01096206996589899, 2.7851505279541016, -0.04865097627043724, 0.002424075501039624, -0.4887422025203705, 0.15567909181118011, -0.010046888142824173, 0.8034497499465942, -0.9484784007072449, 0.0684012845158577, 0.5203989148139954, 1.0777921676635742, 0.35451093316078186, 1.1598389148712158, -0.06667625904083252, -0.011997990310192108, 1.100771188735962, -0.9692460298538208, 0.30948150157928467, -1.3176634311676025, 0.017052732408046722, -0.6375488042831421, -0.1817770153284073, 0.05810091271996498, 1.69991135597229, -0.331475168466568, 0.008825371041893959, 0.02871205471456051, -0.6031066179275513, 0.20707820355892181, 0.22191977500915527, -0.783425509929657, -0.2092481106519699, -0.19923147559165955, 0.6112092733383179, 0.001617938163690269, -3.0294299125671387, 0.05609431862831116, 1.3442659378051758, 0.5891858339309692, -0.032668400555849075, 1.0225502252578735, -0.2995741069316864, -1.1949821710586548, 1.5947015285491943, 0.7580761909484863, -0.8739683628082275, -0.03752163425087929, -2.388561248779297, 0.47159838676452637, 0.21074941754341125, 0.039901312440633774, 1.2070773839950562, -0.023475084453821182, 0.33357739448547363, -0.3893715739250183, 0.0001747157657518983, 0.08127010613679886, -0.01111272256821394, -0.32606467604637146, 1.180887222290039, -0.6553123593330383, -0.06293081492185593, 1.0343056917190552, 0.8591433167457581, 0.3377326726913452, -0.2691999673843384, -0.09687112271785736, -1.7051846981048584, -0.5762043595314026, 0.4255685806274414, 0.41144633293151855, 0.5416308045387268, 0.9621728658676147, 1.2005237340927124, -0.4611043930053711, 1.2212105989456177, 2.7656474113464355, -1.3976178169250488, -0.7305176854133606, -0.13627834618091583, 4.345667362213135, 0.8588675260543823, 0.4203517735004425, 0.16509120166301727, -1.0336804389953613, -1.098864197731018, -0.060302212834358215, -0.053609516471624374, -0.008212813176214695, 0.6971097588539124, -0.7593721747398376, 0.4495835602283478, -0.5561450123786926, 0.24710987508296967, 0.1846720278263092, 0.6320449113845825, -1.1281349658966064, -2.0704288482666016, -0.3308987319469452, 0.1410483866930008, -0.6942422389984131, -1.391321063041687, -0.07737061381340027, 2.3258705139160156, -0.015535159036517143, 0.9790496826171875, 0.520189106464386, -0.25150132179260254, -0.27972719073295593, -0.5602245926856995, -1.5229389667510986, 0.7550855278968811, 1.6652600765228271, -0.03016766346991062, 0.0006767367594875395, -0.8594366908073425, 0.04455774277448654, -0.09479152411222458, 0.973186194896698, 1.348366141319275, -0.16506271064281464, -1.131961703300476, -0.3980337381362915, 0.0006148205720819533, -0.16355010867118835, 0.05527753382921219, -1.5384770631790161, 1.0223804712295532, -1.6471844911575317, -0.3214087188243866, -0.004596418235450983, 1.4820719957351685, -0.2401828169822693, -5.108053207397461, 0.024185366928577423, -0.06678688526153564, -0.2919599711894989, -0.28846609592437744, 0.008478053845465183, 0.17854462563991547, 0.22564004361629486, 1.971907377243042, -0.20225968956947327, 0.01530612725764513, -0.22406482696533203, -0.5043390393257141, -1.9235930442810059, 0.46848011016845703, -0.13244223594665527, 0.12312961369752884, 0.9354552030563354, 0.1947382539510727, -0.5636706948280334, 0.011807562783360481, 0.7996484637260437, -0.03679453581571579, 0.27841126918792725, 0.10498209297657013, 0.14756570756435394, 0.32737651467323303, 0.38731372356414795, -1.6512815952301025, 0.0917157232761383, 0.3205578029155731, 0.16819845139980316, 0.1075766310095787, -0.47915413975715637, -0.0014096536906436086, -0.3333377540111542, 0.001102829584851861, 0.07956001907587051, -0.8122079372406006, 0.9393909573554993, -0.007343002129346132, 0.3597995340824127, -2.0360050201416016, -0.5356742143630981, 0.37273138761520386, -0.03845611959695816, 0.22831997275352478, -0.6041806936264038, -0.12888963520526886, 0.6545290946960449, 0.032697644084692, 1.162537693977356, 0.006406361237168312, 0.010698189027607441, 0.564123809337616, 1.2713786363601685, 0.30067023634910583, -1.2127697467803955, -0.5539247989654541, 0.6929146647453308, 0.36012232303619385, -0.7630536556243896, 0.06845559924840927, 0.16677811741828918, 0.16035525500774384, -0.8678415417671204, -0.2806577980518341, 3.520228147506714, -0.041622668504714966, -0.6047323346138, 0.23698608577251434, 0.5498911142349243, 0.141285240650177, 0.838071882724762, -0.6045666933059692, 0.36331894993782043, -0.10804086178541183, 1.1397411823272705, 0.2721438407897949, -0.5415449142456055, -0.09216650575399399, -0.24070557951927185, -1.5511795282363892, -0.0056340740993618965, -0.011160541325807571, -0.049309585243463516, -0.03763050585985184, 0.0345589779317379, -0.139870747923851, -0.10499238967895508, 0.5953474044799805, 1.3158761262893677, 1.010565996170044, -0.009715728461742401, -0.1743810772895813, 0.11361811310052872, 0.8156095743179321, -0.4788918197154999, 0.76739901304245, -0.01330932043492794, 0.14246544241905212, -0.8439500331878662, 0.012128637172281742, -1.3621459007263184, -2.8299341201782227, -0.13115939497947693, -0.0799701064825058, 1.6641312837600708, -0.03533533215522766, -1.064268708229065, 0.5156079530715942, -1.097025752067566, -0.13516880571842194, -2.715380907058716, -1.0804119110107422, -0.11010771989822388, -0.9979896545410156, -2.393881320953369, 0.4987570643424988, 0.34582802653312683, 0.018539950251579285, -0.0440608486533165, -0.7082942128181458, 0.12245447188615799, 0.37134599685668945, 0.04309619218111038, 0.14665080606937408, -4.508895397186279, 0.35817572474479675, -0.01953529193997383, -0.7390716075897217, -0.23242725431919098, -0.9660768508911133, 1.2947462797164917, -0.49827176332473755, -0.03505131974816322, -0.13243243098258972, -1.4184468984603882, -0.19550743699073792, 0.5239105224609375, -0.25616753101348877, -1.8334578275680542, 0.01635899767279625, -0.789932370185852, 0.2132919430732727, -0.42898187041282654, 0.4379977583885193, -2.2380878925323486, 0.7524566054344177, -0.13490509986877441, -0.1719525009393692, 0.13976897299289703, -1.661723256111145, -0.70692378282547, -0.43224895000457764, -1.3975235223770142, 0.7626209259033203, -4.9633941650390625, -0.0019503505900502205, -0.2936316132545471, 0.24190443754196167, 0.07581067085266113, -0.30544012784957886, 0.07661996781826019, 0.00783896166831255, 0.061091769486665726, -0.2622295916080475, -0.7892767786979675, -0.7704347968101501, 0.016718629747629166, 0.3489557206630707, 0.002275522332638502, -2.1608731746673584, 0.10046076029539108, 0.011493743397295475, -0.3337182104587555, -1.1390957832336426, 0.5675135254859924, 0.06690804660320282, 4.017000675201416, 0.39902713894844055, 1.5611488819122314, 0.16400447487831116, -1.7447350025177002, -0.8585401773452759, -0.4068777561187744, 0.6318552494049072, 0.8561239242553711, -0.07784375548362732, 0.43820229172706604, 1.3827219009399414, 0.7128918170928955, -1.9687004089355469, 0.6472638845443726, 0.0646604374051094, 1.0818226337432861, -0.8439692258834839, 1.1561073064804077, -0.03702741861343384, -0.0030692131258547306, 0.017443286255002022, 0.00497925141826272, -1.4209152460098267, 0.4944832921028137, 1.274570107460022, 1.5892504453659058, 0.31584060192108154, -0.33511143922805786, 0.05450110882520676, 1.2148547172546387, 1.846001386642456, 0.44169801473617554, -0.6706057786941528, 1.6928011178970337, 3.061396360397339, 1.915590763092041, -0.029530830681324005, -0.08352088928222656, -0.04535030946135521, 0.09874098002910614, -0.4134768545627594, 0.07936877757310867, 0.286089688539505, -0.70339035987854, -0.4379580318927765, 0.49896687269210815, -1.3898729085922241, -1.3069881200790405, -0.01425835583359003, 1.1632264852523804, -0.672813892364502, 0.06947503238916397, -0.012170969508588314, -3.359314203262329, 0.0443418063223362, 0.6111133098602295, 0.6974919438362122, 0.3178849220275879, -0.19822172820568085, 0.5877679586410522, 0.10629957914352417, 1.853314757347107, 0.0332651324570179, -1.832156777381897, 0.07861505448818207, -0.32687026262283325, -2.770071506500244, 1.251978874206543, 0.28569626808166504, -0.03416144847869873, 0.10941439867019653, 0.005536437034606934, 1.321520447731018, -1.604689359664917, -0.4638155400753021, 0.305714875459671, -0.3093387186527252, -2.484572410583496, -0.8358362913131714, 0.27220967411994934, -0.749690592288971, -0.0044767968356609344, 0.6576889157295227, -0.4179990291595459, 0.15854597091674805, -0.006208133418112993, -0.6816509366035461, -0.0664084330201149, 0.42087745666503906, -0.007263401057571173, -0.299111932516098, 1.3986220359802246, 0.19467829167842865, 0.09246426820755005, 0.2749685049057007, 0.7600691318511963, -0.6690155863761902, 2.365098476409912, -0.21509721875190735, -0.8034700155258179, 1.6012743711471558, -0.07203543931245804, -0.41078805923461914, 1.2579809427261353, -0.4759361743927002, -1.9586366415023804, -0.09198353439569473, -0.9419670104980469, -0.0433385893702507, 0.36425575613975525, -0.14876167476177216, -0.37028273940086365, 0.11890846490859985, 0.10986059159040451, -0.2516583502292633, 0.19073531031608582, -0.8289161324501038, 0.25767871737480164, 0.22972069680690765, 0.42184269428253174, 1.019594430923462, -0.012777678668498993, 0.019129140302538872, -0.547390341758728, -0.291289746761322, -0.6114746332168579, -3.386019468307495, -1.6945793628692627, 0.21782107651233673, 0.612146258354187, 0.7429884076118469, 1.1392853260040283, -1.0796858072280884, -1.4211668968200684, -0.07655170559883118, -0.977277934551239, -0.09959443658590317, 0.7516334652900696, 0.10190925747156143, -0.19198204576969147, 1.0479779243469238, 0.00343515258282423, -0.10226092487573624, 0.3228381276130676, -0.7197922468185425, -0.3015863001346588, 0.02906220592558384, 1.201721429824829, 0.2274523377418518, 1.1766294240951538, 1.357953429222107, -0.005991993937641382, 0.03714476153254509, -0.9520412683486938, -1.0931183099746704, 0.2565978765487671, -0.05036994069814682, 0.12631462514400482, 0.28803011775016785, 0.3199208378791809, -0.17075923085212708, 1.533562421798706, 0.17461822926998138, -1.2580437660217285, 0.35639432072639465, 0.9847471714019775, -1.7498339414596558, -0.454866886138916, 0.35388442873954773, -0.9866991639137268, -0.5682124495506287, -0.04880153387784958, -0.28750064969062805, -0.005462914239615202, -0.006518437061458826, -0.0465368926525116, -0.6660977005958557, 0.22162474691867828, -0.0550638809800148, -0.12522761523723602, 0.8592281937599182, 0.003998979926109314, -0.6449335813522339, -1.5246284008026123, 0.24461501836776733, -0.4582318067550659, -1.059537649154663, -0.010327461175620556, 0.09047406166791916, 0.32428088784217834, -0.3279738426208496, -0.1631632000207901, -0.22549812495708466, -1.3150447607040405, 0.1207798570394516, 0.8665310144424438, 2.406710147857666, -0.20708535611629486, -1.1535028219223022, 0.013125009834766388, -0.13240325450897217, -0.3386520445346832, -0.2186606228351593, -0.06503055989742279, 0.161582350730896, 0.7881562113761902, 0.015536335296928883, -0.9271783232688904, 1.6914278268814087, 1.097944974899292, -1.3511863946914673, -1.6551529169082642, 0.14246481657028198, -0.40002089738845825, -1.9601656198501587, 0.2125731259584427, -0.12521541118621826, 0.7915989756584167, 0.17826716601848602, 0.015490468591451645, 0.001639704336412251, -1.5745309591293335, -0.14465732872486115, 0.14032581448554993, 0.09944085031747818, -0.3140665888786316, -0.4155944585800171, 0.3234734535217285, -2.1063644886016846, 0.7104099988937378, -0.3083283305168152, 1.4179221391677856, 0.1222604438662529, -0.10680662840604782, -0.10552725940942764, -0.015140218660235405, -0.8946526050567627, -1.2331256866455078, 0.04352665692567825, -0.10108475387096405, 0.04608936980366707, -0.006863259710371494, 0.17886032164096832, -0.46589404344558716, -0.4226588308811188, -0.5203419327735901, -0.39246830344200134, -0.28892022371292114, -0.9018576741218567, -0.05085339397192001, 1.4604674577713013, -0.9993821978569031, 0.8843295574188232, -0.5452278256416321, 1.6736565828323364, -0.06907012313604355, -0.36495429277420044, 3.8035833835601807, 0.054167427122592926, -3.360318660736084, -0.5995015501976013, -0.12991319596767426, -0.17518366873264313, 0.7951256036758423, 2.2912447452545166, 0.6110747456550598, -0.3660663962364197, -1.795137882232666, -0.007043847814202309, 0.08640322834253311, -0.5463374853134155, 0.7124110460281372, 0.9820154905319214, 1.2722601890563965, -2.204098701477051, 0.15260601043701172, 2.273604393005371, 0.016839828342199326, -0.3011200428009033, -0.4285258650779724, 0.7947176694869995, 0.38489145040512085, -0.3771750032901764, 0.6958175301551819, -1.2750484943389893, -0.3540213108062744, 2.720411539077759, 1.0006544589996338, 0.34631574153900146, -0.9457334876060486, 0.4269999563694, -0.4101565480232239, 1.1072841882705688, -0.01772148348391056, -0.23506076633930206, 0.41619154810905457, -0.7593895792961121, 1.084925651550293, -2.0370562076568604, 0.5706972479820251, 0.4764760136604309, -0.9455614686012268, 0.5280964970588684, -0.5202401280403137, 0.14191924035549164, -1.0033565759658813, -1.044510006904602, -0.919456422328949, 0.5038838386535645, 0.27747735381126404, -0.010520219802856445, 0.27194273471832275, -0.15868142247200012, 0.7189852595329285, -0.11424221098423004, -2.780449628829956, 0.38538798689842224, 0.003105048555880785, 2.603768825531006, -2.1574246883392334, -0.23440834879875183, -0.4170418083667755, -0.22336287796497345, -3.576970100402832, -0.6101043224334717, 3.0669233798980713, 0.07253063470125198, 0.04773631691932678, 0.6888378262519836, -1.240960955619812, -1.2644449472427368, 0.3566475808620453, -0.5043532848358154, 2.3305611610412598, 0.34669774770736694, 1.0819950103759766, 0.032014086842536926, 2.793396472930908, 1.7439525127410889, -0.07740039378404617, -0.8065333962440491, 1.985147476196289, -0.8152430653572083, 1.2703123092651367, -1.3014805316925049, 0.4188787043094635, -0.2719445824623108, -2.8943586349487305, -0.13649937510490417, -0.0882922112941742, 6.60936975479126, 0.9946320652961731, -0.2606574296951294, 0.096617192029953, -0.49325037002563477, -0.5400430560112, -0.009287388063967228, -1.464787483215332, 0.07924117147922516, 1.330625057220459, -0.5370360612869263, 0.7180647253990173, 0.0027954347897320986, -0.08421538770198822, -0.8869927525520325, -0.38734185695648193, -0.14393669366836548, 0.6544729471206665, 1.042405605316162, 0.05682733654975891, 0.729483962059021, 0.11024153232574463, 0.1154162734746933, 0.6314876079559326, 0.32033491134643555, -0.1095399260520935, 0.3865657448768616, 0.7846187949180603, 0.6548823714256287, -0.06444215774536133, 1.214150071144104, 0.16173234581947327, -0.06604748219251633, 1.0416113138198853, 0.13215883076190948, -0.28108373284339905, 0.017202727496623993, 0.029341954737901688, -0.03004106506705284, 0.13257260620594025, -3.0299055576324463, -0.5650449395179749, 0.43153443932533264, -0.4496113359928131, -0.12199447304010391, 0.5892691016197205, 0.056198589503765106, -0.31791409850120544, 0.6321799159049988, 0.590384304523468, -0.7007614374160767, -0.0716390535235405, 0.2922089099884033, 0.36590248346328735, -1.0720258951187134, -0.017870767042040825, -0.6477122902870178, -0.08472739160060883, 0.4685978889465332, 0.7646527886390686, 0.05822911113500595, 0.14849604666233063, -0.03379691019654274, -0.23813217878341675, 0.1901257038116455, 0.015709735453128815, -0.4988694489002228, 1.2304949760437012, -0.7978249192237854, -0.11676181852817535, -0.35524338483810425, -0.5169797539710999, 1.9939029216766357, -0.016182519495487213, -2.4092702865600586, 0.07928279787302017, 1.1356788873672485, 4.871156692504883, -0.35799795389175415, 0.06118861585855484, 1.1617039442062378, 0.41982007026672363, 0.3829078674316406, -0.1773621290922165, 0.7958158850669861, -1.6036423444747925, -0.9676975011825562, 0.7408379316329956, 0.13173934817314148, -1.5307350158691406, -0.24089446663856506, -1.9168139696121216, 0.5232866406440735, -2.5427348613739014, -1.212631106376648, 0.473629355430603, 2.7980306148529053, 0.44256070256233215, 0.9092074632644653, 0.14423459768295288, 0.12756772339344025, -1.1260484457015991, -0.8067805171012878, -0.32777711749076843, 1.5252025127410889, -0.16767999529838562, -0.32101306319236755, -0.5457866191864014, -0.3704524040222168, 0.4493934214115143, 0.15093185007572174, 3.7940168380737305, -0.8650869727134705, -0.13424475491046906, -0.06244305148720741, -1.727278232574463, 0.007708861958235502, -0.2360127866268158, 0.12211360037326813, 0.10624384880065918, 0.3377894461154938, -0.5010068416595459, 0.2913239002227783, -0.6816673278808594, -1.0222904682159424, -0.22023195028305054, 0.0871056616306305, -0.3298523426055908, 0.7979735732078552, 0.16915342211723328, 1.5217362642288208, 0.7648516893386841, 0.7864323854446411, -0.28755635023117065, 0.06282682716846466, -0.0667363628745079, -0.46391135454177856, -0.10643066465854645, 0.7540210485458374, 0.24685952067375183, -0.03190919756889343, -0.14967288076877594, -0.5357666611671448, 0.4216802716255188, 0.6142972707748413, -0.061491187661886215, 0.764884889125824, 0.8457245826721191, -0.0071538337506353855, 1.0569654703140259, 3.3659021854400635, -0.8578242063522339, 0.15143391489982605, -0.1263037919998169, 0.13998953998088837, 0.439970999956131, 0.10999879986047745, 0.5657342076301575, -1.7037595510482788, 0.9092315435409546, -0.792205810546875, 0.03973514214158058, -1.1352335214614868, -0.2620096504688263, -2.3773670196533203, -2.197911262512207, 0.2922190725803375, -0.3349912464618683, -0.3992203176021576, 0.6240580677986145, 2.9315035343170166, -0.19554756581783295, 0.5655590891838074, 0.19476407766342163, 0.5981430411338806, -0.1630626618862152, -1.9932913780212402, 0.2517477571964264, -0.3629988729953766, 0.44045373797416687, 0.6937888264656067, 0.318184494972229, 0.4145245850086212, 1.0493816137313843, 1.1016117334365845, 0.2399018108844757, 0.15009234845638275, -0.6420767903327942, -4.581718921661377, 0.04591197893023491, -0.25495803356170654, 0.08075940608978271, -0.10668284446001053, -0.46206212043762207, 0.046537987887859344, 0.8040743470191956, 0.5949806571006775, 0.3906051814556122, 0.47472119331359863, 0.08425541967153549, 0.06836061924695969, 0.9549576640129089, 0.19465960562229156, -0.5738089084625244, 1.216212511062622, -0.4341973662376404, 0.15121789276599884, 1.603643774986267, -0.3808632791042328, -0.32157373428344727, -0.25525569915771484, 1.0376521348953247, -0.030325302854180336, 1.3343700170516968, -3.023583173751831, -0.02206721156835556, 0.7766131162643433, 0.25263845920562744, -0.6565796732902527, 0.3626260459423065, 2.498377561569214, 0.833217978477478, 0.46554821729660034, -1.367872714996338, -0.2032899260520935, -0.33706074953079224, 0.08742654323577881, 0.1153317540884018, -0.1182173639535904, -0.09929439425468445, -0.3428098261356354, 0.28002089262008667, 0.02935617044568062, 1.1243209838867188, -0.3997970521450043, 0.08437050133943558, -2.5066163539886475, -0.5393168926239014, -0.4437031149864197, 0.2533935606479645, -1.1259186267852783, 0.3338997960090637, 0.7129616737365723, 0.1842113733291626, 0.01413776446133852, 0.5505729913711548, 0.9263757467269897, 1.970610499382019, 2.1593704223632812, 0.04716536030173302, -0.8185816407203674, 0.07962837815284729, -0.10625388473272324, -0.1988685429096222, 0.4928176999092102, -0.1430147886276245, 0.29857558012008667, -0.8955185413360596, -0.01239076629281044, -0.04004080966114998, 1.1349666118621826, -0.4935918152332306, -2.129418134689331, -1.8353500366210938, -0.8505134582519531, -0.0038920515216886997, -0.1993437111377716, -0.011672327294945717, 1.2253049612045288, -1.0793694257736206, -0.15811438858509064, 0.04193554446101189, 0.025657232850790024, 0.6550858020782471, -0.08700191974639893, 0.3944384455680847, -0.5978721976280212, -2.8153274059295654, 0.11522302031517029, -0.2719627022743225, 0.49054938554763794, -1.5563195943832397, -5.410928249359131, -0.5115638375282288, -0.15975113213062286, 0.35617852210998535, -0.021466800943017006, 0.0015832976205274463, -0.04700514301657677, -0.42995530366897583, 0.15199948847293854, 0.49232280254364014, 0.08604341745376587, 1.0568430423736572, 0.0064306845888495445, -0.4317452311515808, -0.03945697098970413, -0.06867355108261108, -0.823902428150177, -0.6300344467163086, -1.759040355682373, 0.6854982972145081, -0.5621737837791443, -0.017368724569678307]], "dtype": "float32"}, {"__nd__": [[-1.2743360996246338, -2.388439416885376, -0.11817717552185059, -1.1913487911224365, -1.4929139614105225, 1.556423306465149, 2.214675188064575, 2.9950451850891113, 1.782516360282898, -0.12779316306114197, 0.5338742733001709, 1.1983481645584106, -0.9904366135597229, -0.09936592727899551, -0.43468740582466125, 0.5192558765411377, 1.0040876865386963, -2.0445966720581055, -0.48070961236953735, -0.5913347005844116, 0.28052616119384766, -0.6920210123062134, 0.079867422580719, -0.23718863725662231, 0.8200076818466187, -0.5252102017402649, -0.3197481632232666, -0.2983364462852478, 5.10694694519043, 0.9110914468765259, -2.1256580352783203, 1.5379250049591064, -0.6562764048576355, 0.5379800796508789, 2.149134635925293, -1.9598466157913208, -1.1413142681121826, -0.5027393698692322, -2.698133945465088, 0.6314245462417603, -0.8843014240264893, 1.5937063694000244, 1.415995478630066, 1.8043832778930664, -2.2152607440948486, -0.41300004720687866, 0.8344854712486267, 0.9885780811309814, -1.2020525932312012, -1.490411639213562, 1.3493623733520508, 1.7218008041381836, -1.6028262376785278, -2.4970505237579346, 0.0964721143245697, -1.6235361099243164, 3.331615924835205, 0.014274239540100098, -0.11395293474197388, 1.525388240814209, 1.2365505695343018, -1.1251155138015747, 0.1218111515045166, 0.7588615417480469, -1.9386029243469238, 0.07456231117248535, 0.17985695600509644, -0.2064937800168991, 1.1517882347106934, -1.0218896865844727, -0.2534576654434204, -1.0347447395324707, -1.0737813711166382, 1.5477503538131714, 0.9018601179122925, 1.2960495948791504, -0.11828362941741943, 1.0460805892944336, 1.5718564987182617, 1.1177905797958374, 2.7136831283569336, -1.0398520231246948, -0.911763072013855, -1.3192803859710693, 2.7236595153808594, 0.4620450735092163, -2.380920886993408, 0.2749888002872467, 0.7852624654769897, -1.1468336582183838, -0.6948137283325195, 0.39596113562583923, 0.6922613978385925, 0.01629909873008728, 0.8670727014541626, 0.9963404536247253, 2.922179937362671, -1.0295543670654297, -1.561394214630127, -0.05867069959640503, 0.31051313877105713, -0.45282822847366333, 1.8062421083450317, -1.885737657546997, -2.6141042709350586, 0.6165676116943359, 1.7822449207305908, 2.7695798873901367, -1.3761017322540283, 1.53392493724823, -2.0249557495117188, 3.2928130626678467, 0.37944281101226807, -0.18989235162734985, -2.2534353733062744, 0.44543999433517456, 0.03995054215192795, 0.8632517457008362, -0.6513227820396423, 0.8310562968254089, -0.9756026268005371, 1.7587320804595947, -2.170106887817383, -0.6199814081192017, 3.619608163833618, 0.11907985806465149, -0.8692037463188171, 1.0909054279327393, -0.32980412244796753, -0.6364949941635132, -0.5545973777770996, 0.8586738705635071, 0.11488741636276245, 0.7974148392677307, -0.7642459869384766, 1.9080593585968018, -0.05494034290313721, -0.4100167751312256, -0.9577457904815674, -0.7132924795150757, -0.14974069595336914, 0.12432575225830078, 0.17509007453918457, -1.289297342300415, -1.4252314567565918, -0.7019006609916687, -0.7828364968299866, -1.6738625764846802, 1.5301272869110107, 2.329514980316162, -1.5062596797943115, -1.1716753244400024, -0.1775389313697815, 1.102941870689392, 2.211972713470459, 1.902976393699646, -0.6798831820487976, 1.3373363018035889, -2.0163207054138184, -0.9606301188468933, -0.06634920835494995, -0.19525763392448425, 0.24825549125671387, -0.9270159006118774, -0.746180534362793, -1.2168035507202148, -0.24585115909576416, 2.7846715450286865, -2.0791730880737305, -1.5749175548553467, 1.1516120433807373, 3.889881134033203, 2.1226813793182373, -0.45178884267807007, -1.8973667621612549, 0.765038013458252, 0.7096501588821411, -0.8503516316413879, 1.8472506999969482, -0.7160007953643799, -2.0992817878723145, -1.0891029834747314, 0.25032827258110046, -0.636877715587616, 3.190998077392578, -0.7477654814720154, 0.2241116166114807, 2.3127925395965576, -2.62262225151062, -0.15642070770263672, -0.17161822319030762, 0.2820875942707062, -0.383358359336853, -0.28414374589920044, 2.1379973888397217, 1.542690396308899, 0.2903282046318054, 0.5417041778564453, -1.323309063911438, 1.3180204629898071, 0.24371641874313354, 0.2803994417190552, 0.9820475578308105, 1.3900822401046753, -0.2362900972366333, -0.44935595989227295, -1.2792925834655762, 0.7201650738716125, 0.6125590801239014, 2.303131580352783, 1.669323205947876, -0.5474945306777954, -3.273794651031494, -0.5034839510917664, -0.34808117151260376, 0.2011096477508545, -1.4379938840866089, 2.2242400646209717, -1.758034110069275, 1.8678162097930908, 0.0373988151550293, -0.36494189500808716, 2.1806650161743164, -1.1615303754806519, -0.4924801290035248, 0.14100098609924316, 0.11692564189434052, -0.14664384722709656, -1.8854986429214478, -0.12080377340316772, 1.7777202129364014, 2.8653101921081543, 0.24342375993728638, -0.18593576550483704, -1.9625669717788696, -1.6769524812698364, 1.9203083515167236, 0.9578549265861511, 1.385278344154358, 2.1516847610473633, -1.6116232872009277, -0.5723894834518433, -0.3485504686832428, -2.846710443496704, -0.577736496925354, 0.9959152936935425, 1.977292537689209, -0.029279515147209167, -1.0517072677612305, 0.4812033772468567, -0.30009064078330994, 0.7597026824951172, 1.6408816576004028, -0.6709022521972656, -1.0338350534439087, -1.7179949283599854, 0.7002060413360596, -1.4865694046020508, 0.3299916386604309, 0.1545337438583374, 1.40952467918396, 1.3292772769927979, -1.569455623626709, 0.6608642339706421, 1.376356601715088, 1.3781921863555908, 3.0541090965270996, -0.5214494466781616, -3.9536898136138916, -0.8441922664642334, 1.8621360063552856, 1.0958656072616577, 0.08791583776473999, -2.4053711891174316, -1.7586545944213867, 0.1583516001701355, 0.6582517623901367, 1.8646091222763062, -0.7336587309837341, 0.8817334175109863, -0.9767583608627319, 0.5793238878250122, -0.16023457050323486, 1.1947834491729736, -0.8616628646850586, -0.5006512403488159, 0.742636501789093, -0.9071592688560486, -1.8074054718017578, 1.4901776313781738, -1.939686894416809, 1.537384271621704, -0.6162080764770508, 0.4006717801094055, 0.24459515511989594, -1.0775560140609741, -0.8525766134262085, 0.8714060187339783, -0.0019714832305908203, -2.44053316116333, -0.7065994739532471, -2.892047882080078, -1.6914353370666504, 1.3005231618881226, 2.1064677238464355, -0.9896947145462036, -1.8155651092529297, -0.34129372239112854, 0.07850044965744019, 0.2581534683704376, -1.166691541671753, -0.032684385776519775, -0.42020314931869507, -0.7443641424179077, -3.5179214477539062, 0.36703720688819885, 0.9733952879905701, -0.971594512462616, 0.8137224316596985, -0.9826234579086304, 0.44386959075927734, 0.0022514760494232178, 4.019680500030518, -0.2943245470523834, -0.7347636818885803, 0.8446139693260193, 1.2639083862304688, 0.8442861437797546, -1.54900062084198, 0.20160925388336182, -1.1040645837783813, 3.9688589572906494, -1.9510024785995483, -0.005719393491744995, 0.12627121806144714, 0.20135465264320374, -0.6508027911186218, 0.615099310874939, -0.4564352035522461, 1.8087655305862427, -2.297776699066162, 0.24955374002456665, 3.0010299682617188, -0.4459846615791321, 0.941480278968811, 3.242189407348633, -0.7842453718185425, -0.8193116784095764, 1.1697165966033936, -3.141839027404785, -0.8084871768951416, -0.4972909390926361, -1.095213770866394, 1.1697120666503906, 1.5744843482971191, 0.8203166723251343, -0.9697662591934204, -0.5366864800453186, -0.0736127495765686, -1.2275631427764893, 2.32022762298584, 1.4913448095321655, 1.8930721282958984, -0.6802353262901306, -0.6035316586494446, -1.0211751461029053, -1.8122738599777222, -0.8375024199485779, 0.22812896966934204, 0.18370595574378967, 0.1324230432510376, 1.9542404413223267, 1.3002140522003174, -0.12235301733016968, -0.19561433792114258, 0.1362784206867218, -1.4019557237625122, -1.3031120300292969, -0.37211549282073975, 0.8577010631561279, -1.5650651454925537, -0.25520065426826477, 0.4902000427246094, 1.143768548965454, -0.18826250731945038, 2.4280166625976562, -0.3204333186149597, 2.9169576168060303, -0.14503297209739685, -1.2531814575195312, -1.2142497301101685, 0.47899729013442993, -0.09735649824142456, 0.9975816011428833, 2.2704052925109863, -1.2603224515914917, 0.28541696071624756, 1.4502952098846436, 0.8535517454147339, -0.054563045501708984, -0.6434857845306396, -1.063612699508667, -0.05010542273521423, -2.1725406646728516, -0.4588701128959656, 0.3616752624511719, -2.639547109603882, 1.005851149559021, -0.4946119487285614, -2.8169116973876953, 1.3913512229919434, 0.6053525805473328, 0.4090704917907715, -1.1863124370574951, -1.5869793891906738, 0.3908159136772156, -1.321907877922058, -0.9847234487533569, 0.48939818143844604, 0.10218527913093567, -2.4119718074798584, -0.13255101442337036, -0.9203764200210571, 0.5357190370559692, -0.5138128995895386, 0.527767539024353, 0.9799962043762207, -0.15274037420749664, -0.12548135221004486, -0.6652945876121521, -0.10627126693725586, -1.5878992080688477, -0.676230788230896, -0.33774518966674805, 2.681028127670288, 1.0853652954101562, -0.07109054177999496, 0.4909028708934784, -0.9543931484222412, 0.060697704553604126, -1.6316993236541748, 1.5749800205230713, -1.011492133140564, -2.5697824954986572, -0.4300376772880554, 0.41283783316612244, 2.8806204795837402, 1.043503761291504, -0.6471789479255676, -2.2645137310028076, -0.9780126214027405, 3.0822248458862305, -0.18286670744419098, 0.545289158821106, 0.6875181794166565, 1.351393461227417, 0.8829838633537292, -0.05813312530517578, 2.274230480194092, -1.7146961688995361, 0.32738351821899414, -2.238016128540039, -1.2355585098266602, 2.59672212600708, 1.115922451019287, 0.4043824076652527, 1.9551504850387573, -1.77423894405365, 0.7535996437072754, 2.0901081562042236, -0.2230890393257141, 0.16705703735351562, -0.7152765989303589, 1.9334027767181396, -0.6320351362228394, 1.2017549276351929, 2.15228009223938, -0.8656426668167114, 0.19235414266586304, 3.2907395362854004, -0.9398750066757202, 0.7775670289993286, -0.6419599056243896, -0.937335729598999, 0.27201923727989197, 3.6721582412719727, -1.3302080631256104, -0.7426714301109314, 0.6977092027664185, -1.3058216571807861, -1.1598278284072876, -0.4814445376396179, 1.137420415878296, 1.0373964309692383, -0.534587562084198, -0.6465339064598083, 2.145632743835449, -1.2289435863494873, 1.3586070537567139, -0.6225323677062988, -1.1793222427368164, 1.0043902397155762, -2.3591554164886475, -0.16353555023670197, -0.39917874336242676, 1.0569936037063599, 1.1041934490203857, -0.5511559247970581, -1.4401781558990479, -2.7802348136901855, 2.0489118099212646, 2.8095169067382812, -0.12999585270881653, 1.7599021196365356, -1.6178405284881592, 1.3342819213867188, 0.6416361331939697, -0.21451520919799805, 0.8350483775138855, -1.6629347801208496, 0.15322351455688477, -0.8491690754890442, 1.5211701393127441, -0.09667348861694336, -1.3384110927581787, -2.358820676803589, -2.609161853790283, -1.0421380996704102, 1.100833535194397, -1.7944530248641968, 0.5971516370773315, -0.03338426351547241, -0.9026212692260742, -1.6130729913711548, 0.6098585724830627, -0.6849486231803894, 0.22239720821380615, 0.30115798115730286, 0.31968557834625244, 0.9592303037643433, -0.10650515556335449, 2.462630271911621, -1.525345802307129, 0.3172893524169922, 0.49041062593460083, -2.2344818115234375, 1.5976810455322266, -0.9231491088867188, -2.750056028366089, -3.6136057376861572, 1.4254416227340698, 0.04476940631866455, -0.6726736426353455, 0.4478180706501007, -0.23604202270507812, 2.1414999961853027, 3.4851651191711426, -1.4237592220306396, 1.3764712810516357, -0.04086538776755333, -3.2303552627563477, 0.8449229598045349, -0.6262184381484985, -0.11205703020095825, 4.010913372039795, -1.1606531143188477, -1.3279495239257812, -1.0919177532196045, 1.396470308303833, 0.8048320412635803, 0.8522606492042542, -0.5040449500083923, -0.08087577670812607, -1.0221529006958008, -0.06449207663536072, 0.5364886522293091, 2.2683088779449463, -1.222993016242981, 1.4160633087158203, 0.5707672834396362, 2.813586711883545, 3.6112022399902344, -0.5408811569213867, -1.0874236822128296, 0.2737564146518707, 2.380815029144287, -0.7939755320549011, 1.4904768466949463, 2.7700228691101074, -1.3007540702819824, -2.288316249847412, 0.7598994970321655, 0.8170215487480164, -0.47424495220184326, 0.2485576868057251, 4.512686729431152, 0.019092649221420288, -0.23138880729675293, 0.34566357731819153, 1.5297996997833252, -1.7944622039794922, 0.6372754573822021, -0.0067313313484191895, -0.9922195672988892, 2.113450050354004, -0.46521228551864624, -0.38185346126556396, -2.154184103012085, -1.624948501586914, 1.5343294143676758, -2.2004318237304688, -0.9861481189727783, -1.5041677951812744, -0.7195745706558228, -1.8271160125732422, 0.24104639887809753, -2.0897300243377686, 1.2590727806091309, 1.987306833267212, -3.0445263385772705, -0.2164512574672699, 0.6532553434371948, 3.135715961456299, -0.02707993984222412, -0.12092244625091553, -0.47958144545555115, 1.3684334754943848, -1.392570972442627, -1.9064680337905884, 2.2358694076538086, 0.6956486701965332, -3.1663408279418945, 2.3202459812164307, -0.5879786014556885, 0.21945369243621826, 0.3981740176677704, 1.248958706855774, -0.0786038339138031, -1.2913997173309326, -1.3578072786331177, -0.966715931892395, -2.366464138031006, -0.7835037112236023, 3.5916976928710938, -2.4973690509796143, 0.9915366768836975, 1.3336542844772339, 1.3919800519943237, 0.6652728319168091, 3.457641839981079, -1.1394965648651123, -0.8853966593742371, 1.0330312252044678, -1.3620446920394897, 0.05203652381896973, -1.8506121635437012, 1.0868974924087524, -1.9657914638519287, 0.3436189293861389, -0.48753634095191956, 0.17158688604831696, -1.046968936920166, 3.08134388923645, -0.8552433252334595, 0.3947845697402954, -0.26867806911468506, 0.7475966215133667, 3.3476243019104004, -2.0977942943573, -3.093231678009033, -1.9601887464523315, -1.2175089120864868, 0.6761618256568909, 2.1831672191619873, 1.9051908254623413, -0.7779940962791443, 1.1256226301193237, 0.6886153817176819, 0.22049038112163544, 0.31430327892303467, 1.8390774726867676, 0.13771294057369232, 1.3822178840637207, -0.27543193101882935, 1.0797982215881348, 0.7103850841522217, 1.1980067491531372, 1.9252049922943115, 1.4272569417953491, 0.42168667912483215, 0.6971653699874878, -0.13590914011001587, -0.45182013511657715, -0.052559733390808105, -3.6682722568511963, -1.9926190376281738, -1.3758537769317627, 1.2323462963104248, 1.0617196559906006, 0.8283639550209045, 0.7685258388519287, -2.773608684539795, -1.8264836072921753, 1.5099440813064575, 2.3492608070373535, -0.4249783158302307, -2.7450878620147705, 0.21628493070602417, 0.6275955438613892, 0.7817707061767578, -0.8886016607284546, 0.9332762956619263, -2.558259963989258, -1.6289722919464111, 2.813905715942383, -0.16583728790283203, 2.8868584632873535, -1.298781394958496, -0.5905836224555969, -0.5420140027999878, -0.5306146740913391, 0.7991249561309814, 0.4601748287677765, -0.8039865493774414, 0.38529932498931885, 0.6237908601760864, 0.5425626635551453, -1.720165729522705, -1.5833557844161987, -1.0473484992980957, 2.9669859409332275, -0.8502455353736877, 1.1452217102050781, -0.4512341320514679, -0.8469935655593872, 1.1824084520339966, -0.36218470335006714, -0.40926241874694824, 0.821836531162262, 1.1266146898269653, -2.013528823852539, -1.3787345886230469, 0.42645177245140076, -0.849456787109375, -1.1023004055023193, -0.22250798344612122, 0.8921945691108704, 0.05591363459825516, -0.22914963960647583, 1.9122295379638672, -0.17627227306365967, -0.08738470077514648, -2.0634260177612305, -1.350588321685791, -1.7980128526687622, -1.588033676147461, -1.911645531654358, -0.4734685719013214, -1.195199728012085, -0.4795038402080536, -1.5087001323699951, 0.3276313543319702, 2.9544596672058105, 1.5651317834854126, -0.021883249282836914, 0.3072689473628998, -1.8722286224365234, 0.836402177810669, -0.041380882263183594, -0.767040491104126, 0.08668386936187744, -1.6478911638259888, -0.014744043350219727, -2.0914478302001953, 1.1999895572662354, -1.5605943202972412, -0.8146905303001404, 1.6311259269714355, 2.5405101776123047, 1.6421644687652588, -0.30084100365638733, -0.28534919023513794, 1.2093815803527832, -0.40320974588394165, 1.3849174976348877, 3.6090593338012695, -2.025517225265503, -1.1301175355911255, 0.3194117546081543, 0.5348340272903442, -1.0555455684661865, -0.4672885537147522, -0.07902646064758301, -1.0197844505310059, 1.627562403678894, 0.9958518147468567, 2.363176107406616, -0.4792136549949646, -1.1357338428497314, -0.655411422252655, 1.3737999200820923, 0.6098067760467529, 1.1414973735809326, -0.1055733859539032, -2.3829097747802734, 2.100640296936035, -0.23963770270347595, -0.21865686774253845, -0.3423917591571808, 1.1312036514282227, -2.5160467624664307, -2.3876538276672363, 0.3558996319770813, -0.8173122406005859, -0.7113016247749329, 0.2256636917591095, -0.6131492257118225, -1.5435848236083984, 1.3367226123809814, 0.5144309401512146, -0.007044672966003418, 0.075141042470932, -0.29564952850341797, 0.6943491697311401, 1.9181933403015137, 1.2697947025299072, 0.42332667112350464, -0.925356388092041, 0.6114692687988281, 0.4430341422557831, 0.6474618911743164, -0.17624568939208984, 0.7869625091552734, 0.6927672624588013, -0.9349075555801392, -0.28428196907043457, -1.5811355113983154, -2.3011422157287598, 0.9166489839553833, 0.4497680962085724, -0.7962440252304077, -1.697605848312378, 2.9623610973358154, 0.42393767833709717, -0.4299509823322296, 0.48685598373413086, -0.004611454904079437, 0.35972684621810913, -0.6687831282615662, 2.7072644233703613, -2.0946483612060547, -0.05983155965805054, -0.681298017501831, 1.3796910047531128, 0.13307291269302368, -0.6915056705474854, 2.0665388107299805, -2.5437192916870117, -0.5713292360305786, 1.3522392511367798, 0.1733582317829132, 0.5633171200752258, 0.9304145574569702, 1.7802271842956543, -2.097588539123535, 0.07042276859283447, -2.226619243621826, -0.2035207748413086, -0.2157665491104126, 0.7681887149810791, -0.41045165061950684, 1.2043534517288208, -0.5257626175880432, -1.2830013036727905, -1.025223731994629, 1.3667185306549072, -1.9084930419921875, 0.23364025354385376, 1.6428718566894531, -1.7689796686172485, -0.18771308660507202, 1.168418288230896, -1.279662013053894, -1.8376191854476929, 0.8647249341011047, 1.961477279663086, 0.17989152669906616, -3.8493404388427734, -0.23913605511188507, 1.739965796470642, -1.2429341077804565, -0.577033519744873, -0.37145233154296875, -1.1688610315322876, 0.12199956178665161, -1.4199789762496948, -2.1824557781219482, -1.444485068321228, 0.30285659432411194, 1.359694242477417, 1.4372029304504395, 1.4404640197753906, 0.1415938436985016, -1.25927734375, 3.3270437717437744, -0.1973031461238861, 0.3822002410888672, 3.37616229057312, 1.6893213987350464, -1.3757474422454834, 1.5007045269012451, 1.0178160667419434, 1.2652922868728638, 0.8656437397003174, 1.2931625843048096, -1.363059401512146, 0.4754517674446106, 1.4670227766036987, -1.3647162914276123, -1.1549813747406006, 2.1638453006744385, -0.3296477794647217, -2.5682220458984375, 1.0778907537460327, -0.5917185544967651, -1.9166501760482788, 1.3150484561920166, -0.6055267453193665, 2.7630672454833984, -0.07997921109199524, 1.078522801399231, -2.7840380668640137, -1.535956621170044, 0.2430688440799713, -0.4861869812011719, 0.9777120351791382, -0.19992320239543915, -0.8405114412307739, -1.6807405948638916, -0.4123777151107788, -2.241199016571045, -0.8398916721343994, 1.2974228858947754, -1.0280768871307373, -0.4753774404525757, 1.9849215745925903, -1.2610759735107422, 1.3320345878601074, -0.5279613733291626, -1.2508381605148315, -1.3347179889678955, -0.35529381036758423, 2.497955083847046, 1.5024298429489136, -0.42865920066833496, 0.48471301794052124, 0.015946924686431885, -0.04237934947013855, -3.341768264770508, 2.642688512802124, 0.06770926713943481, 1.3586050271987915, 0.7217031121253967, 1.0848844051361084, 0.12943699955940247, 0.9909225106239319, 0.711066722869873, -0.6054990887641907, 0.6183236241340637, -0.2245909720659256, 0.762876570224762, 1.2783657312393188, -2.8685731887817383, -1.689467191696167, -1.0654369592666626, 1.2703114748001099, 1.295254111289978, -0.5355262756347656, -0.900718092918396, 0.839208722114563, -0.15345066785812378, 1.7255735397338867, 2.023207187652588, -0.4030379056930542, 0.12577903270721436, 0.5845411419868469, 0.20330184698104858, -0.1666194200515747, 0.7709731459617615, -0.16848042607307434, -1.0288057327270508, 0.8431715369224548, -1.0979747772216797, -1.1832661628723145, 0.3079698085784912, -2.2439002990722656, 1.6475977897644043, 1.3187086582183838, -1.7747269868850708, -1.2630854845046997, -0.0862080454826355, -0.3294854462146759, -2.9134421348571777, 3.1097662448883057, 0.8229396939277649, -0.40589725971221924, -1.165454387664795, -0.9095242023468018, 1.3392845392227173, -1.051389217376709, -0.6429446935653687, 0.7924463748931885, 0.9726347327232361, 0.7038851976394653, -2.135956287384033, -2.538576126098633, 0.29320526123046875, 0.19956202805042267, -2.6095874309539795, -2.0985517501831055, -1.7555835247039795, -1.1950385570526123, -1.6002075672149658, -0.3599729537963867, 0.7431365251541138, 1.0648322105407715, 1.8020380735397339, -1.0532515048980713, 0.6467281579971313, 0.7780591249465942, -0.6596850156784058, -0.6036710739135742, -1.2855174541473389, 0.10612961649894714, 0.8410512208938599, 1.9937589168548584, 1.1084331274032593, -2.1938443183898926, -1.2068345546722412, 1.9127025604248047, -1.0351005792617798, -1.2204322814941406, 0.22022967040538788, 0.3399569094181061, 1.9875192642211914, 0.8630397319793701, -2.755688428878784, -0.5837840437889099, 1.276863694190979, 0.21803069114685059, -1.4641985893249512, 1.8788580894470215, -0.3872569799423218, 2.8393759727478027, 1.9488296508789062, 1.1531955003738403, 1.0834020376205444, 0.8025287389755249, 0.41625726222991943, -2.1384997367858887, -1.1201303005218506, -0.7762512564659119, -1.0856553316116333, 2.7058985233306885, -1.5500521659851074, 1.6075488328933716, 0.9130228757858276, 1.0493584871292114, -1.6590869426727295, 1.3379414081573486, 1.5751063823699951, -1.9426205158233643, 2.232820510864258, 0.10862749814987183, -1.2594902515411377, -1.0140920877456665, 0.5817680358886719, 0.8367184996604919, 0.91815185546875, -0.9011582732200623, 0.13918113708496094, 2.9527177810668945, 3.61549711227417, -3.042696714401245, -0.955427348613739, -0.19402474164962769, 1.3472846746444702, -0.057689979672431946, -1.4927594661712646, -0.6774462461471558, 2.4681217670440674, -1.6847127676010132, -1.9255268573760986, -1.7247283458709717, 0.5046643018722534, 2.1295933723449707, -1.6080358028411865, -1.3433107137680054, 0.06153033673763275, 0.4893367290496826, 1.2172125577926636, -0.5726358890533447, 0.6435397267341614, -2.2823328971862793, -1.31303071975708, 2.262971878051758, 0.5936899781227112, 1.0423228740692139, -0.18797524273395538, -1.6634366512298584, 1.3196687698364258, 1.1877151727676392, -0.19781669974327087, 0.12466394901275635, 0.4361687898635864, 0.27187249064445496, 0.09170174598693848, 2.24385929107666, -1.0364041328430176, 2.0382790565490723, 2.180846691131592, -0.9658861756324768, 3.3661444187164307, -0.34646517038345337, -0.03339719772338867, 1.23482346534729, -1.0197410583496094, 2.6643128395080566, 1.2133886814117432, -0.07096493244171143, -0.1509343981742859, -1.7257587909698486, -0.007438719272613525, 1.7598553895950317, -1.972798228263855, -0.7185418605804443, 0.13258475065231323, -0.35020846128463745, 0.023507699370384216, -2.055434465408325, 1.6629459857940674, 1.1821249723434448, 0.9067585468292236, 0.13064181804656982, -1.0696730613708496, 0.2661413848400116, -0.24431085586547852, -0.17232827842235565, -0.3805271089076996, 2.510030508041382, -0.7358536124229431, 1.3445160388946533, -1.1667735576629639, -2.039741039276123, 2.8478026390075684, -0.7600029110908508, -0.7687122821807861, -1.3202600479125977, -1.0014300346374512, -0.04938890039920807, -1.4762890338897705, -0.4497348964214325, -2.379594326019287, 3.333881378173828, 1.7385210990905762, 2.140766143798828, -2.8481388092041016, -0.8003564476966858, 1.0634442567825317, -0.7403603792190552, -0.4929850101470947, -0.7120633721351624, 0.36581820249557495, 1.1246819496154785, -1.3099015951156616, -1.1673957109451294, 0.4335675835609436, 0.29780471324920654, 1.0309233665466309, 1.5472180843353271, 2.030888080596924, -1.3757201433181763, 0.8697096109390259, -1.5273463726043701, -0.46296072006225586, -2.1448824405670166, 0.19739419221878052, 1.6722679138183594, 0.07208865880966187, 3.3030343055725098, -1.5939090251922607, -1.3775379657745361, 0.1035912036895752, 0.8219400644302368, -0.48381200432777405, 4.0514235496521, 1.251689076423645, -0.19851261377334595, 0.09590908885002136, 1.6383646726608276, -1.6683073043823242, -0.47319382429122925, 1.5127665996551514, 0.7717088460922241, 1.9234333038330078, -3.0909340381622314, 0.5719633102416992, 0.49060946702957153, 0.37563982605934143, -0.05389547348022461, 1.1890885829925537, -0.9430378079414368, 1.440940499305725, 0.8861397504806519, 0.730349063873291, -1.9218072891235352, 0.06852089613676071, 1.1082605123519897, 0.6858110427856445, -0.4291210174560547, 0.5543672442436218, 1.0468741655349731, -0.22623173892498016, 0.47335973381996155, 0.07421267032623291, 0.23591405153274536, 0.9768869876861572, 1.4219987392425537, -0.6878889799118042, 0.9597439169883728, 1.0437783002853394, -1.2393747568130493, -0.006147848442196846, 0.046843647956848145, 0.3689649701118469, -0.6277868151664734, -0.26800793409347534, 0.1902921497821808, -1.1872683763504028, -1.4980285167694092, -2.3308749198913574, -1.9563994407653809, 1.0966594219207764, -0.6395218372344971, -1.110169529914856, -1.8388620615005493, -0.6096080541610718, -0.4065053462982178, 0.2931567132472992, 0.4985864758491516, -1.2531299591064453, 0.4403091073036194, 0.4468248188495636, -0.1278427541255951, 0.18831413984298706, 0.23230409622192383, 0.3359142541885376, -0.5647861361503601, 0.9153323173522949, -0.022872507572174072, -2.4500484466552734, -0.6808815598487854, 1.240152359008789, -0.600114107131958, -0.05591711401939392, -0.7362096905708313, -1.7512307167053223, -1.3926723003387451, 1.5574359893798828, -1.5154712200164795, 0.7821578979492188, -0.37728607654571533, -0.016406241804361343, -1.055220127105713, 0.6373772621154785, -0.04565918445587158, 0.5197922587394714, 1.508004903793335, 0.2857566475868225, -0.47967100143432617, 0.8661890029907227, -3.327929735183716, 0.050898611545562744, 0.7431484460830688, 3.958934783935547, 1.0731743574142456, -0.12112756073474884, -1.659016489982605, 2.714632987976074, -1.0147099494934082, 0.4512706398963928, -0.746230959892273, 0.5742079019546509, 1.2159745693206787, -0.5558831095695496, 0.5174649357795715, 1.412603735923767, 2.2791619300842285, -1.21771240234375, -0.005759239196777344, 1.9541453123092651, -3.5940465927124023, -0.7060486078262329, 0.020175091922283173, -1.591628074645996, -0.09043705463409424, 0.651860773563385, 1.901181936264038, -0.6944820880889893, 1.3735334873199463, -0.16806796193122864, 0.7894593477249146, -1.721603274345398, -0.8327822685241699, 2.1591436862945557, 1.3613113164901733, 0.03357747197151184, 0.7529692649841309, -0.4883354604244232, 0.9638978242874146, -0.7663908004760742, 0.0634947419166565, 2.727980375289917, -0.40949010848999023, -1.0190974473953247, 1.959367036819458, -0.12913838028907776, 2.754626750946045, -0.7936428189277649, 0.48072466254234314, -1.6888635158538818, -0.08823007345199585, 0.8778716921806335, 3.629098415374756, 1.5810215473175049, -0.7610466480255127, -0.5584580302238464, -1.0057111978530884, 0.4557913541793823, 1.495764136314392, 2.402587413787842, -3.401386022567749, -0.9591321349143982, 1.184258222579956, 0.32727840542793274, -0.42818114161491394, -1.5676789283752441, -3.2188429832458496, 2.169724464416504, -1.3592677116394043, -1.7026673555374146, 1.730271339416504, 0.6258877515792847, 0.0028510689735412598, 1.1645934581756592, -0.713703989982605, 0.27504438161849976, -0.5351768136024475, 1.043522834777832, 2.9850316047668457, 1.3352361917495728, -0.5547087788581848, -0.18280184268951416, 0.1668289750814438, 0.23866960406303406, 1.7346892356872559, -1.0963479280471802, 1.482649803161621, -1.6465234756469727, -2.632841110229492, -1.8403151035308838, -2.361585855484009, 2.4227757453918457, -1.5360689163208008, -1.5283609628677368, 1.938501238822937, 0.017230510711669922, 1.510590672492981, -1.7082459926605225, 1.6469120979309082, 1.4569463729858398, -1.5012693405151367, 0.41285425424575806, -0.952059268951416, 1.6008986234664917, 1.308842658996582, 1.1529759168624878, 0.17793488502502441, 1.9451725482940674, 0.408088743686676, -1.8572347164154053, -1.3285210132598877, 0.8710567355155945, 1.769068956375122, 0.8825450539588928, -2.064457893371582, 1.6263564825057983, 0.6546871066093445, -1.179051160812378, 0.8803757429122925, -0.4865434169769287, 0.4238112270832062, 0.0954342782497406, -1.631739616394043, 1.5608091354370117, 1.3678522109985352, -0.19471672177314758, -0.5986100435256958, 0.8683538436889648, -0.9690777063369751, 1.1889123916625977, -1.9973523616790771, 0.8253666162490845, -1.512225866317749, -1.2012639045715332, -1.2369074821472168, -0.09032636880874634, -0.042679429054260254, -1.361076831817627, -1.2322710752487183, -0.19383853673934937, 0.9461197853088379, 1.6161432266235352, -0.2114417552947998, 1.7297956943511963, -1.2589563131332397, -0.3943783640861511, 0.46420735120773315, 0.3467192053794861, 0.34280842542648315, -1.5462158918380737, 1.1002938747406006, 1.2751308679580688, -0.009399503469467163, -2.51800537109375, -1.2099831104278564, -1.492556095123291, -2.706648349761963, -1.309805154800415, -1.2673108577728271, -2.0977749824523926, 0.19904613494873047, 3.36531662940979, 0.633012056350708, 1.7345167398452759, 0.9861167073249817, 0.5092519521713257, 0.17251846194267273, 0.6699931621551514, 0.3880515694618225, 1.0524561405181885, 0.9811967611312866, -0.5493078827857971, -0.2180527001619339, -1.0131456851959229, -1.3522478342056274, 1.3021799325942993, -2.0713562965393066, 0.5220279693603516, -3.0789761543273926, -2.1055707931518555, -0.3489511013031006, -0.7005591988563538, 0.006039977073669434, 1.0923906564712524, 0.3269280791282654, -1.5565075874328613, -0.39301246404647827, -1.39874267578125, 2.2795004844665527, -0.4405374526977539, 1.477284550666809, -1.852373719215393, 2.0481762886047363, 0.6116970777511597, -0.7500039339065552, 0.7406400442123413, 1.1473971605300903, 1.9244177341461182, 0.02931380271911621, 3.1658935546875, -0.3064347505569458, 0.06381034851074219, 0.6877481937408447, 0.04329755902290344, 0.051468148827552795, 2.0942037105560303, -0.6246898174285889, 0.005899548530578613, 0.5634126663208008, -1.1553024053573608, -0.3261604905128479, 1.34766685962677, -1.5952402353286743, 0.056821346282958984, -1.0344929695129395, 2.739041328430176, -0.4590144157409668, -1.2013145685195923, -0.5540006160736084, 1.7247244119644165, 1.0111452341079712, 1.0106098651885986, 0.540355384349823, 0.8336969017982483, 2.983205795288086, 1.341463565826416, -0.7646661996841431, 0.746803879737854, -0.027072250843048096, -2.178171396255493, -1.085245132446289, -1.827293872833252, -2.47702956199646, -1.018904685974121, 1.8456453084945679, -0.6434827446937561, 0.2370111346244812, -0.2580406665802002, -2.496304988861084, 3.074859142303467, 0.1663031131029129, -0.00421716645359993, -0.32198458909988403, 4.561530113220215, -0.7258524894714355, -1.082822322845459, 1.4506690502166748, -3.8263649940490723, 2.3304646015167236, 0.344599187374115, -1.4825420379638672, 0.46253228187561035, 2.1676621437072754, -0.2820754051208496, 0.14696946740150452, 1.1225676536560059, -2.0960419178009033, 1.215235710144043, 0.4222731590270996, 0.5750130414962769, 2.708068370819092, 1.2102855443954468, 1.6760814189910889, 1.8188873529434204, 0.7963887453079224, 1.5714085102081299, 1.076874017715454, -1.965996503829956, -0.36855337023735046, 2.050985336303711, -0.3886221647262573, 0.6454405784606934, -0.6904128789901733, 1.7193078994750977, 0.5688067674636841, 2.3987104892730713, 0.9033094048500061, 0.49408864974975586, -2.5851776599884033, 0.48587706685066223, -1.3352351188659668, 2.2475476264953613, 1.1404879093170166, -0.22121058404445648, 0.1431497484445572, -1.3841370344161987, -0.2905540466308594, 1.0933330059051514, -0.0744728147983551, 1.6876602172851562, -0.5006911158561707, -0.6381112337112427, 1.2815825939178467, 2.3922948837280273, 0.537703275680542, -1.2972710132598877, -1.5106804370880127, 1.0241384506225586, 2.7377443313598633, -1.852248191833496, 0.022203445434570312, 0.5007274746894836, 0.8560210466384888, -0.5279947519302368, -0.12657546997070312, -0.6891002058982849, -2.9221444129943848, 1.7234652042388916, 0.05726468563079834, 1.7009434700012207, 3.832340717315674, -1.5611622333526611, -0.9261291027069092, -0.7240694165229797, 1.9689347743988037, 0.26374632120132446, -1.1927504539489746, -1.10056734085083, -0.9930345416069031, 0.6943903565406799, 1.0081818103790283, -0.17768073081970215, 1.4105315208435059, -1.771970510482788, -1.761532187461853, 0.8945364952087402, 1.297372579574585, 3.229098320007324, -0.40205270051956177, 0.5565596222877502, -3.149688720703125, -0.1456398367881775, -1.3482755422592163, 4.274324417114258, -0.10417279601097107, 0.09845829010009766, 1.1927238702774048, -0.7196968197822571, -0.6596987247467041, 0.2594183385372162, 2.3630213737487793, 0.846940815448761, 0.7716480493545532, 0.37783321738243103, 2.270482063293457, -1.2367199659347534, -0.16976076364517212, 0.5847262144088745, 2.334409236907959, 0.0710662305355072, -2.106205940246582, -0.11218991875648499, -0.005182623863220215, 0.5140202641487122, -2.734649181365967, -0.9004757404327393, -2.0804548263549805, 0.060776486992836, 0.7575072050094604, -0.05355179309844971, 1.1002776622772217, 2.1236352920532227, -0.24293643236160278, 0.500266432762146, 2.2819437980651855, -0.07170915603637695, 0.12538082897663116, 0.38502389192581177, -0.26316553354263306, 0.3602266311645508, -0.6016512513160706, -1.21201491355896, -0.5607883334159851, -0.900985836982727, -0.41194218397140503, -1.7435481548309326, -0.9759672284126282, -1.4653489589691162, 0.005771994590759277, -0.1235632672905922, -0.9313362836837769, 1.8181507587432861, -0.5314983129501343, 1.3729832172393799, 0.5423986315727234, 0.9776984453201294, -1.7631491422653198, -0.9323914051055908, -0.6528264880180359, -0.5276566743850708, -0.16393521428108215, 0.9933515787124634, -1.0674612522125244, 0.03663337230682373, -2.5561726093292236, 1.878467321395874, 0.9570770263671875, 0.3056122064590454, 1.3020029067993164, -2.8976645469665527, 0.33703115582466125, 0.42012277245521545, 0.8171777725219727, 0.6721265316009521, -0.002705499529838562, -0.19122153520584106, 1.3818633556365967, 1.0479800701141357, 1.4175156354904175, 0.38508927822113037, -3.320427417755127, -0.11828712373971939, -1.8714065551757812, 0.5226976275444031, 3.0865087509155273, -3.429147720336914, -2.1074962615966797, 2.1077985763549805, -1.3329359292984009, -1.179502248764038, 2.4407856464385986, -1.1160078048706055, -1.4403486251831055, 3.494701623916626, 0.1432906687259674, 1.020167589187622, -1.1986066102981567, 1.2914057970046997, -1.1544705629348755, 0.14267680048942566, 2.9075510501861572, -0.8963300585746765, -0.07699063420295715, -1.846204161643982, 1.3675402402877808, -1.1547448635101318, -1.4968626499176025, -3.3162741661071777, 1.2041927576065063, -0.6086211800575256, 1.7809393405914307, -0.28976932168006897, 0.11227178573608398, 2.320215940475464, -3.015124797821045, -3.678523063659668, -1.816660761833191, 1.1745312213897705, 0.45804524421691895, -0.370492160320282, 2.3840930461883545, -2.118587017059326, -2.310908079147339, 1.129351019859314, 3.4823741912841797, 1.4934011697769165, -0.23440492153167725, 0.6354544162750244, -0.6422652006149292, -1.7826423645019531, -0.9799407720565796, -1.0020678043365479, -0.17701071500778198, 0.8433206677436829, -1.2373021841049194, -0.2116348147392273, -2.241395950317383, -1.1295183897018433, 1.283153772354126, 1.2354933023452759, 0.6081444025039673, 1.5197422504425049, -0.1963750272989273, -0.04324057698249817, -1.8881947994232178, -1.2146683931350708, 2.3359978199005127, -1.053061842918396, 2.5040929317474365, 0.0035991929471492767, -0.31024235486984253, 2.1724023818969727, -0.36402055621147156, 0.19474616646766663, 1.9088647365570068, -1.8864033222198486, -1.9818280935287476, -0.30660828948020935, -0.31122279167175293, -1.4131009578704834, -2.7791457176208496, -0.2894097566604614, -1.0175496339797974, 1.2509568929672241, -0.39311644434928894, -0.7075079083442688, 2.0123953819274902, -1.399744987487793, -0.06927365064620972, 1.4869945049285889, 1.9753060340881348, -0.38175317645072937, -0.24040234088897705, 0.5116927027702332, 1.0166223049163818, -2.689232349395752, -2.009810209274292, 1.5319979190826416, 0.6748602986335754, 0.13723987340927124, 0.5537673234939575, 1.3020851612091064, -2.287447929382324, -0.7458593249320984, -2.9639055728912354, -0.8995457887649536, -1.1089057922363281, 0.08228647708892822, -4.274558067321777, -2.4528684616088867, -1.570403814315796, -0.5791898965835571, -0.024863511323928833, 0.2632354199886322, 2.496802806854248, -0.12360918521881104, 0.8524858355522156, -0.4608820676803589, -0.11647239327430725, 1.3693172931671143, 1.12837553024292, 0.3168922960758209, -0.7499462962150574, -1.594102144241333, -1.7256057262420654, -0.36892127990722656, -1.0752418041229248, -0.35427212715148926, -0.6211885213851929, -0.3207473158836365, 1.1029129028320312, 1.5691251754760742, -0.13178634643554688, -0.39916351437568665, 1.83240807056427, 0.7810994982719421, -0.47074174880981445, 1.4266563653945923, -1.6662899255752563, -1.7263509035110474, -0.7004896402359009, 0.8778215646743774, 0.4390876889228821, 2.039436101913452, -0.9081096649169922, -3.3047003746032715, 0.2709693908691406, 1.2766036987304688, 1.4652667045593262, 0.6098121404647827, -1.0143474340438843, -0.07976675033569336, -0.13410437107086182, 3.31925368309021, -3.6479177474975586, 0.11102868616580963, 0.4316950738430023, -0.8878335952758789, -0.17707771062850952, 0.317413330078125, 0.04025879502296448, -1.3813270330429077, -0.3438173830509186, -1.5895676612854004, 2.320253610610962, 0.14683756232261658, 0.09675751626491547, -1.4556257724761963, -0.2875651717185974, -1.6623873710632324, 0.5402954816818237, -0.19170206785202026, 0.10059183835983276, -1.3495792150497437, 0.9817588329315186, 0.13019424676895142, 1.446958065032959, -1.7652232646942139, -0.35410428047180176, 1.105907678604126, -0.23449289798736572, 0.45809125900268555, 0.5542888045310974, -1.1798017024993896, -2.5953567028045654, -0.9430435299873352, -1.443679928779602, -0.16370654106140137, 0.6343973875045776, 0.5995815992355347, 0.012100696563720703, 0.026608906686306, 0.9061713218688965, 1.1804300546646118, 4.106956481933594, 2.20554256439209, -1.6693415641784668, 0.40525519847869873, 0.84662926197052, -1.0060608386993408, 0.873382568359375, -0.9108937978744507, -0.9947344064712524, 2.1927943229675293, 0.732213020324707, -1.1688194274902344, 0.687814474105835, 0.6762575507164001, 1.9788731336593628, 0.24384410679340363, 0.4618077874183655, 0.9811691045761108, -0.957033634185791, -2.4010324478149414, 1.4118468761444092, 0.3282575011253357, 0.9962841272354126, 1.6062105894088745, -1.208024024963379, -0.019715309143066406, -0.29470396041870117, -0.6211689710617065, 0.27530479431152344, 0.3866160213947296, 0.06043629348278046, 1.1619073152542114, -1.2925819158554077, -2.618596315383911, 0.44731855392456055, 0.44857174158096313, 0.0923498272895813, 0.735592782497406, -1.2834298610687256, 1.7376505136489868, -1.596860408782959, 0.08770114183425903, 2.1123738288879395, 0.38891932368278503, -1.108508586883545, -2.08675217628479, -0.6282275915145874, 2.2017605304718018, 2.3558459281921387, 1.0907915830612183, 2.357154130935669, -0.25010037422180176, 0.21817950904369354, 0.8412808775901794, 1.511216163635254, -1.045739769935608, -0.20237129926681519, 0.6042971014976501, -0.13435488939285278, -0.47327643632888794, -1.5814366340637207, 0.4557601511478424, 0.6215085983276367, 0.7990860939025879, 0.05107736587524414, -0.8561805486679077, 0.2203957438468933, -2.4551658630371094, -1.5365066528320312, -1.6458977460861206, -0.06863182783126831, -0.8127288818359375, 1.403471827507019, 2.2035367488861084, -1.0506726503372192, -2.2606494426727295, -0.21486949920654297, 2.173912525177002, -2.481811761856079, 0.7042484879493713, -0.18651899695396423, -0.5308700203895569, -0.5761052370071411, -1.409796953201294, 1.9301897287368774, 0.04198795557022095, 1.524242877960205, 0.8280231952667236, -2.5927562713623047, -0.05425262451171875, 0.405162513256073, 0.7582077980041504, 0.5619664192199707, -4.079592704772949, 1.7820074558258057, -0.2099546194076538, -0.514866292476654, 0.30629444122314453, -0.7047128081321716, -0.1598188877105713, -1.4768474102020264, 1.1240146160125732, -0.716895580291748, -2.494983673095703, 1.2016031742095947, 0.2701238989830017, -2.190436363220215, -0.48020753264427185, -1.370876669883728, -0.24422767758369446, -2.0773184299468994, -0.690054714679718, 0.8942259550094604, 1.76923406124115, 1.9459052085876465, -1.0865511894226074, 1.9785380363464355, 0.8536612391471863, -2.008282423019409, -0.12300962209701538, 0.8362600803375244, -1.4498090744018555, 0.19872130453586578, -0.13760870695114136, 1.0623369216918945, 0.3356558680534363, -1.2144373655319214, -2.0097436904907227, -1.582605242729187, -3.162102699279785, 0.15722018480300903, -1.1663172245025635, -1.8695950508117676, -1.0269631147384644, -0.5254796743392944, -1.8742742538452148, -0.12207742035388947, 0.4978722035884857, -1.4308947324752808, -2.7663826942443848, 0.4734157621860504, 0.2874489426612854, 1.4452941417694092, 1.832845687866211, 0.5761493444442749, -0.8287843465805054, -2.013504981994629, 0.8597524762153625, -1.1361910104751587, 0.24997468292713165, -0.3611585199832916, -1.7543418407440186, -2.0436952114105225, -1.3396978378295898, 2.885948657989502, 0.5197626352310181, 0.13505005836486816, -0.9637781381607056, -1.1138792037963867, -1.7801438570022583, 0.0561942458152771, 2.5440175533294678, 1.651456594467163], [-0.44199854135513306, -3.1146178245544434, 0.4000648260116577, 0.8317486643791199, -1.475020170211792, -3.6903743743896484, 1.365501046180725, -1.2915215492248535, -0.9311650991439819, -1.9201622009277344, -1.061245322227478, -2.032038688659668, 0.28908199071884155, 0.884581446647644, -1.130709171295166, 1.8008944988250732, 2.052016496658325, 1.0218318700790405, 1.1162863969802856, -0.8090473413467407, -0.16288256645202637, 0.5418037176132202, 3.1700992584228516, 3.200960636138916, 1.5583491325378418, 0.10125581175088882, -0.7838190793991089, -1.814570426940918, -0.7924516201019287, -0.025368213653564453, 0.6959745287895203, 0.8639021515846252, -2.0341670513153076, 1.5391819477081299, 0.14025843143463135, -1.8584344387054443, -0.3323376774787903, 1.3278887271881104, 0.492266982793808, 0.7792810201644897, 0.5271764993667603, -0.44925227761268616, 0.1829676628112793, -0.8530929088592529, -0.1531798541545868, -1.336295485496521, 1.2727360725402832, 0.3194023370742798, 0.48296380043029785, 0.5630149245262146, 1.8768610954284668, 0.6433952450752258, 1.2473266124725342, -0.8076204061508179, 0.7297601699829102, -1.6435086727142334, 1.5233101844787598, 1.6290287971496582, -0.021495357155799866, -0.5264546871185303, 0.491126149892807, -0.22777360677719116, -0.6509127616882324, -2.5213658809661865, -0.43144452571868896, 1.6059664487838745, 0.7283607721328735, 1.1463695764541626, -0.5628269910812378, 0.9446894526481628, 0.6761950254440308, -1.9684689044952393, -0.9722632169723511, -0.44021832942962646, -1.7419174909591675, 0.8777069449424744, -1.2512142658233643, 0.9055271744728088, -0.7438086867332458, -0.5510708093643188, -2.079495429992676, 0.21271371841430664, 2.0852391719818115, -0.5299656391143799, -3.097895860671997, -0.31478679180145264, 0.978484034538269, 1.0586515665054321, -1.2628949880599976, 0.6607570052146912, 0.22970154881477356, 0.7948597073554993, -0.12601019442081451, -1.3275343179702759, -0.04188418388366699, 0.8776576519012451, 0.4430966377258301, 2.3715131282806396, -2.2337963581085205, 0.326540470123291, 1.0458495616912842, 3.0238265991210938, -2.1184699535369873, 0.10293370485305786, 0.6773660182952881, -1.1117284297943115, -1.2233173847198486, 0.11483144760131836, -0.5368363261222839, 0.5682358145713806, 1.037438988685608, 2.921351909637451, -0.15596939623355865, 0.029886841773986816, -2.464203357696533, -2.71061372756958, 2.6934690475463867, 0.5036792755126953, 0.39079153537750244, -0.3955717384815216, 0.5519569516181946, 1.888217568397522, -1.3801255226135254, -1.8562818765640259, -2.2428741455078125, 0.8018167614936829, 2.1102631092071533, -1.5510101318359375, 0.8941112160682678, -1.0096086263656616, 1.0349903106689453, -2.7989799976348877, 0.8536741733551025, 1.420837163925171, 1.4558448791503906, -0.46812641620635986, -0.8298380374908447, -0.09609588980674744, 1.0207180976867676, 0.4345957934856415, -0.33173853158950806, -0.1017865538597107, 0.6948181390762329, -1.2152475118637085, -0.015333324670791626, -1.4958710670471191, -0.12239968776702881, 0.4831608831882477, -1.7050702571868896, -1.9322993755340576, 4.0680317878723145, -2.545807123184204, -1.4759844541549683, -0.44140803813934326, -1.0003488063812256, -0.8942211866378784, -0.6786848902702332, 1.1191668510437012, -0.4779990613460541, 2.3598198890686035, -2.237448215484619, 0.7739989161491394, 0.7449097037315369, 0.40739989280700684, -0.28410738706588745, 0.30718857049942017, 1.2276766300201416, 0.20646655559539795, 2.4722230434417725, 1.3304823637008667, 0.7366529703140259, -4.045044898986816, -0.08792304992675781, -2.0474777221679688, 2.4501113891601562, -0.8789091110229492, 1.846714973449707, -0.329687237739563, 0.2704622149467468, 2.103635787963867, -1.4500818252563477, 2.2329137325286865, 3.1549909114837646, -1.5665479898452759, 2.4764232635498047, 1.2607147693634033, 0.48368117213249207, -0.7599375247955322, 0.9889893531799316, 0.20687389373779297, 1.331770420074463, -3.4358859062194824, -2.812354326248169, -2.528604507446289, -0.23369044065475464, -0.4332655072212219, -2.776611089706421, -0.026679277420043945, -0.4777164161205292, -2.179804563522339, -0.41411226987838745, -0.8102810978889465, 0.4982960820198059, 0.725563645362854, -2.0352745056152344, -0.6952387094497681, 1.189988374710083, -0.6778035759925842, -0.5524238348007202, 1.5639605522155762, 1.456634521484375, -0.1452392190694809, -1.6109158992767334, -0.35470303893089294, 0.12689915299415588, 0.6434228420257568, 1.9719626903533936, -0.46615177392959595, 1.0533500909805298, 0.7050532102584839, -0.3517055809497833, -2.9045968055725098, 0.30597984790802, 0.05171510577201843, -2.613264322280884, -0.17103421688079834, -1.6357470750808716, 0.5163604021072388, 0.9732001423835754, -2.3752949237823486, -1.2457191944122314, -1.6172311305999756, 0.3225356340408325, -1.1931474208831787, -0.6539818644523621, -1.4665372371673584, -0.7987092733383179, 0.9536223411560059, -1.6207449436187744, -0.4345175623893738, -1.661885380744934, -0.3543093204498291, -0.1608296036720276, -0.36801236867904663, -0.36347347497940063, -2.6684279441833496, -0.10589201003313065, 0.07659509778022766, 0.1607845276594162, -1.283063292503357, 2.5605015754699707, 1.662225365638733, 1.064894437789917, 1.2835228443145752, 0.6416866779327393, 0.4664488732814789, 0.7246992588043213, -0.4786704182624817, -1.4040920734405518, 0.06764145940542221, -0.816230297088623, -1.9208941459655762, 0.6980170607566833, 0.9043455123901367, -1.9235268831253052, 3.7428078651428223, -1.4488139152526855, -0.23982995748519897, 2.517836809158325, -0.34619665145874023, -1.4128413200378418, 0.8365278244018555, -0.7738137245178223, 1.0122394561767578, 1.5333318710327148, 0.007308542728424072, 0.42747724056243896, -0.4994155764579773, 0.07927540689706802, 0.8952938318252563, 1.4538812637329102, 0.6636111736297607, -1.5954718589782715, 2.1614603996276855, 1.4424749612808228, -1.7865989208221436, -1.7626745700836182, 0.3951897919178009, -0.4299120604991913, 1.573578953742981, -3.4648566246032715, -0.3760947585105896, -0.07321804761886597, 1.1452990770339966, -0.5399428606033325, 1.7082443237304688, 1.5679969787597656, 0.6029840707778931, -1.4851248264312744, -0.9161050319671631, 1.2353764772415161, 1.435017466545105, 1.3029875755310059, 1.443037509918213, 0.25950273871421814, 1.0659205913543701, -1.601233959197998, 1.2960435152053833, -3.1547951698303223, 0.10966942459344864, -0.16413891315460205, -3.1720049381256104, 0.011837657541036606, 1.8153448104858398, -1.7677264213562012, 1.226248025894165, 0.24153999984264374, -1.3170608282089233, -2.1665353775024414, -4.3683905601501465, 1.4112827777862549, 1.5385322570800781, 0.07201486825942993, 2.281525135040283, -1.2377517223358154, 0.8771895170211792, -1.2616877555847168, -0.6142339110374451, -2.596909761428833, -1.9162238836288452, 0.6970700621604919, 0.47799152135849, 0.7795658111572266, 0.5577587485313416, -1.678950548171997, 0.8463735580444336, 1.915513277053833, 1.6931045055389404, 0.8033658862113953, -0.08180993795394897, -0.7786229252815247, -1.465232014656067, 2.3745615482330322, -0.9350112676620483, -0.07781553268432617, 0.6447206139564514, 1.610884666442871, 2.325603723526001, 0.20033568143844604, -1.3692634105682373, -3.1590404510498047, 0.5918313264846802, -0.24344097077846527, -1.5048731565475464, -1.3920844793319702, -0.7043105959892273, 0.36484014987945557, -0.7653331756591797, 0.40623947978019714, -0.8973620533943176, 1.034501075744629, 2.8154873847961426, 1.1579880714416504, 0.32105308771133423, -0.5401120781898499, -0.2764456272125244, 0.7489426732063293, 0.6740548610687256, 1.814206600189209, 1.9085420370101929, -2.910248279571533, 0.9677501916885376, -3.298330783843994, -0.3448601961135864, 1.8343346118927002, -1.553963303565979, -0.4114934802055359, 0.5746504664421082, 1.339435338973999, 2.1296606063842773, -1.8359007835388184, -0.3214876055717468, 0.25029247999191284, 0.6154354214668274, 1.3986889123916626, 0.5338741540908813, 0.5852231383323669, 1.1705385446548462, -1.4904617071151733, -1.6196537017822266, -1.5295606851577759, 0.7357025146484375, 0.8118085861206055, -1.8139985799789429, 3.058037519454956, 2.009812593460083, -0.5664342641830444, 0.22150465846061707, 0.9130819439888, 2.141632318496704, 2.289059638977051, 0.7802163362503052, -1.3489313125610352, 0.09166038036346436, -2.556692123413086, 1.3944592475891113, -1.7872437238693237, -1.464331865310669, 0.20437300205230713, -1.136653184890747, 2.7280755043029785, 0.7814407348632812, 0.5555702447891235, -1.0813236236572266, -1.4919230937957764, 1.5789659023284912, 0.682624340057373, 0.05076548457145691, -0.9988248348236084, 2.496506690979004, 1.8598213195800781, 0.73619145154953, 1.1022813320159912, 0.8065959215164185, 0.25974100828170776, -0.19007684290409088, -1.117781162261963, 0.40807852149009705, 0.4800746440887451, 1.2669727802276611, -1.412992238998413, -1.5068490505218506, 0.8577824234962463, -0.5463725924491882, 1.2968562841415405, 1.3027656078338623, -0.6679660081863403, 1.7176225185394287, 0.37834280729293823, 1.5160797834396362, 0.600641131401062, -0.5055116415023804, -0.39770156145095825, 0.8369301557540894, -1.641968011856079, -1.5531996488571167, -0.5153805017471313, -2.0828261375427246, -0.5961042046546936, -0.5904380083084106, -1.7748745679855347, -1.6814584732055664, -0.10950100421905518, -1.535973072052002, 0.13319504261016846, 3.0072693824768066, -1.6500638723373413, -2.283719301223755, 1.981238842010498, 1.1428500413894653, -1.2568238973617554, -3.5250842571258545, -0.05056619644165039, -0.37158462405204773, -0.7862452268600464, 1.7612214088439941, 2.3190603256225586, -1.2381863594055176, 1.3002912998199463, 2.880436658859253, 0.5798157453536987, 1.0196577310562134, 0.7983459234237671, -3.152740955352783, 1.2694190740585327, -0.4521414041519165, 0.41709476709365845, 1.2643065452575684, -0.5827797651290894, 0.8925696015357971, -0.1344669759273529, -2.223836660385132, -0.3077150583267212, -0.11558288335800171, -0.32441744208335876, 1.0765498876571655, 0.7209876775741577, 0.0042465925216674805, 0.1819249987602234, 1.4105280637741089, -0.8697707653045654, 0.7162586450576782, 3.4223544597625732, -1.6900432109832764, -2.2496955394744873, -2.4576587677001953, 1.6058814525604248, -0.5135218501091003, 0.4136149287223816, 2.212433099746704, 0.3741471767425537, -1.0762951374053955, 0.6840188503265381, -0.6712344288825989, 0.5473045110702515, 2.0453126430511475, 0.8729746341705322, 0.11272051930427551, -0.9201399683952332, 0.4853311777114868, -0.9557192325592041, -0.46228694915771484, -0.15352189540863037, 0.9160662293434143, 1.089370846748352, 0.03555595874786377, 0.7747312188148499, 1.9607106447219849, 1.2979958057403564, -1.454967737197876, 0.5487192273139954, 0.2150949239730835, 1.27328622341156, 2.5264837741851807, -1.7606961727142334, -2.188281536102295, 0.038920581340789795, -0.8064799308776855, -0.7954111099243164, 1.3159602880477905, -0.9998862147331238, -0.3879739046096802, 1.6868476867675781, 1.652998685836792, 0.5752251744270325, -0.6174193024635315, 0.0964009165763855, -1.3706015348434448, 1.132164716720581, -3.4401774406433105, -0.08126738667488098, 0.7544711828231812, 2.781377077102661, -0.8951958417892456, 0.6736571788787842, -1.474172830581665, -0.38941943645477295, 0.5791981816291809, -0.5234183669090271, 0.3871048390865326, -0.19895493984222412, -1.0715737342834473, -0.08111723512411118, 1.00535249710083, -1.0264692306518555, -2.527925968170166, 0.29355958104133606, 1.5411376953125, -1.819157600402832, 2.0123043060302734, -0.9259067177772522, 2.753643035888672, -1.01755690574646, -1.3363604545593262, 1.566569209098816, 0.32890915870666504, 0.15950441360473633, -0.5270112752914429, 1.4838558435440063, 0.9243446588516235, 2.221268653869629, -0.8127136826515198, 1.024308443069458, -2.495276689529419, -0.8807881474494934, -0.2846590280532837, 0.5942358374595642, 1.1855080127716064, -0.9489660859107971, 2.4648640155792236, 0.41565781831741333, 0.41077134013175964, 0.7503693103790283, 0.34741026163101196, 1.1967520713806152, 1.6994246244430542, 2.8943653106689453, 0.9129633903503418, -0.06920862197875977, 0.2355404943227768, -0.23385486006736755, -2.6773881912231445, 0.44544947147369385, 1.6552395820617676, -1.7283014059066772, -1.5326588153839111, 1.2593021392822266, -0.07253417372703552, 0.6145578026771545, 3.415041446685791, -1.9684401750564575, 0.39191925525665283, -1.7323694229125977, -0.8176836967468262, -0.035675495862960815, -1.5378029346466064, 2.2333900928497314, 1.7882702350616455, 0.6391820311546326, 0.07576797157526016, 1.2497735023498535, 0.1550232172012329, 0.5261067748069763, -1.0630892515182495, 1.9804933071136475, 1.4927432537078857, 0.3258711099624634, 1.5191829204559326, -4.611154556274414, 2.708220958709717, -0.5043078064918518, 0.689399242401123, 0.12793412804603577, 0.6708409786224365, 1.5659382343292236, -0.5017577409744263, -1.6196956634521484, 3.936497926712036, -2.231794834136963, 0.6541706323623657, 0.32559460401535034, -0.9028784036636353, -0.8312804102897644, -2.816084623336792, 1.4383516311645508, 0.5290703773498535, 1.0949325561523438, -0.8549609184265137, 0.1794431060552597, -0.6046380400657654, 2.003286361694336, -1.3715530633926392, 1.406069278717041, 1.5583726167678833, -1.6929340362548828, -1.8232353925704956, -1.7111103534698486, 1.1193913221359253, 2.171658992767334, -2.2246973514556885, -0.21379238367080688, 1.123439073562622, 1.7662352323532104, -2.850965738296509, 1.152889370918274, 1.9270069599151611, -0.21076124906539917, -0.7697370052337646, 1.690248727798462, 0.6109890937805176, -1.6610838174819946, -1.036412000656128, -3.5032033920288086, 2.4430007934570312, 1.9568490982055664, -0.25419145822525024, -1.1113163232803345, -1.444455862045288, 0.03240945562720299, -0.15497927367687225, 2.7962374687194824, -0.0008215010166168213, 0.8201063871383667, -2.4745430946350098, 1.3478742837905884, -1.517501711845398, -2.653289794921875, 0.611617386341095, -1.705479621887207, -2.5007517337799072, 1.1622124910354614, 0.9420115947723389, -2.2311201095581055, -0.23999136686325073, -0.43707120418548584, 0.672592043876648, -0.739543080329895, -0.2232769876718521, 0.9049431085586548, 0.7264821529388428, -2.2148630619049072, 2.8450214862823486, 0.8815258741378784, 2.055495500564575, -0.6566042900085449, 0.9035506844520569, 0.8709314465522766, 0.295817494392395, -0.13458943367004395, -0.5451734066009521, 1.7398970127105713, -0.5896434187889099, -2.66792631149292, 0.4351637363433838, 0.6123113632202148, -1.0881576538085938, 0.8799413442611694, 0.9211428165435791, 2.5074424743652344, -0.7890167236328125, -1.857433557510376, 0.4852418303489685, -3.9402971267700195, -1.1111401319503784, -0.8662511110305786, 1.1098825931549072, -1.2085851430892944, -0.007467091083526611, -0.27028918266296387, 1.0780444145202637, -3.4150424003601074, 2.4839625358581543, 0.5154955387115479, -2.006443500518799, -1.4707809686660767, 2.01019549369812, -0.5775984525680542, 0.22842013835906982, 2.793262481689453, -0.08608847856521606, -0.7223979830741882, 1.7365233898162842, 1.860860824584961, -1.5870451927185059, -0.4997301697731018, 0.5661106705665588, 0.16887176036834717, 0.9421637058258057, 1.8314571380615234, 2.0859274864196777, -0.504474937915802, 4.452038764953613, -1.449791669845581, 1.4060183763504028, 1.6914184093475342, -2.7080326080322266, 1.3587836027145386, 2.101100206375122, 0.48050737380981445, -1.6617763042449951, -0.6904817223548889, -2.6886038780212402, -2.7590837478637695, -2.072321653366089, -0.3337985873222351, 0.8222885131835938, 0.48759207129478455, -0.22314637899398804, -0.9678961634635925, -0.8719605803489685, 0.4710524082183838, -0.5669772028923035, -0.9805368185043335, 1.454304575920105, 0.5106619000434875, -0.37399452924728394, -1.1654108762741089, -1.5929207801818848, -1.9032769203186035, 0.8819413781166077, 1.1958028078079224, -0.27035441994667053, 0.8506196737289429, -1.9430673122406006, -0.3631610870361328, -0.7200426459312439, 1.565438151359558, 0.5068140029907227, 1.3097550868988037, 1.826743483543396, -0.04731428623199463, 0.38868725299835205, 0.09861516952514648, -0.9627556800842285, 2.0274293422698975, -0.891524076461792, -0.7667536735534668, 1.0816911458969116, 0.732189416885376, -0.6655423045158386, 2.067793369293213, 0.34938740730285645, -1.970234751701355, -0.6794600486755371, -0.7003432512283325, -2.246281385421753, 0.700946569442749, -0.704318106174469, 0.7958387732505798, -0.15461164712905884, 0.1214495301246643, 3.274447202682495, 0.05958474427461624, 0.38890132308006287, 0.7481334805488586, -1.696476936340332, 2.5463833808898926, -0.37775665521621704, -2.7556378841400146, 0.6625239849090576, 2.2392940521240234, -1.9787118434906006, -0.5644606351852417, 1.313107967376709, -0.17250250279903412, 1.0409858226776123, -2.2313904762268066, 3.6752796173095703, -1.538909912109375, 0.6546628475189209, 0.19202619791030884, 2.4515976905822754, -2.0058796405792236, -1.5526314973831177, -0.5162787437438965, -0.3106745481491089, -0.620457649230957, 0.02181553840637207, -1.6386653184890747, -0.8636565804481506, 0.9912292957305908, -0.04934579133987427, 0.6328368782997131, 2.0121779441833496, -1.6409382820129395, 0.38484489917755127, -2.600222110748291, 0.32530152797698975, 0.7618516683578491, -0.31341254711151123, -1.028241515159607, 1.0302342176437378, -1.8314464092254639, -1.174829125404358, 1.0923633575439453, 1.200807809829712, -1.2436524629592896, -0.9429296255111694, -1.3103699684143066, 0.2482132911682129, 1.6575617790222168, 0.9420259594917297, -1.1164324283599854, 1.6192808151245117, -0.8374117612838745, 0.8343263268470764, -1.8147284984588623, -0.818314790725708, -1.3915462493896484, 1.198635458946228, 0.9908403158187866, -1.9562442302703857, -1.066821813583374, -1.2161492109298706, 0.6580196619033813, -1.0342206954956055, 1.3080782890319824, 0.5698634386062622, 1.3191568851470947, 1.459159016609192, -0.5571414232254028, 0.8235065937042236, -0.9173665046691895, -1.0830954313278198, 0.4184291958808899, -0.5383632183074951, 0.42451024055480957, 3.0934512615203857, -0.686260461807251, 0.15629589557647705, 0.7031474113464355, -0.995062530040741, -0.8094813227653503, 1.983881950378418, -1.5000476837158203, 0.03379637002944946, -1.017303228378296, -0.6257369518280029, 0.6534908413887024, -1.271821141242981, -0.33616185188293457, 2.653136730194092, 1.5967496633529663, 2.2204947471618652, -0.6138547658920288, 0.16388624906539917, 0.1665002703666687, 1.943244457244873, -0.01849483698606491, 1.119267463684082, 1.5832126140594482, 0.25747692584991455, 0.3771863281726837, -0.4932260513305664, -1.7796657085418701, 0.6996517777442932, -1.440561056137085, 2.484010696411133, 0.9730355739593506, -0.5632603764533997, 3.471696615219116, -0.7540169954299927, 1.8927526473999023, -2.7506728172302246, -2.084453582763672, -1.75998055934906, -1.1199105978012085, -2.640697479248047, -1.8441317081451416, -0.982629120349884, 1.8334991931915283, 0.42224329710006714, 0.8516740798950195, 0.6676842570304871, -1.1419246196746826, 0.6971707940101624, 0.007720351219177246, -1.6462113857269287, 2.157315492630005, -1.3983640670776367, -1.29562509059906, 0.876777708530426, 0.06040757894515991, 0.8810761570930481, 0.23638856410980225, -0.5911035537719727, -0.7352851629257202, -1.4587684869766235, -0.32406163215637207, 0.7355006337165833, -0.2608727216720581, 0.9447342753410339, 0.7862569093704224, 1.7644758224487305, -0.12881052494049072, -0.6617996692657471, -1.4688613414764404, 0.6071727871894836, -1.4137060642242432, 0.6143515110015869, -1.172515630722046, 0.411861777305603, 0.05914844572544098, 2.244126796722412, -1.616377592086792, 1.9903640747070312, -2.209394693374634, 0.43517833948135376, -2.95658540725708, 0.29998254776000977, 0.11563757061958313, -0.12243771553039551, 1.6054279804229736, 0.2977696657180786, 1.933268666267395, -0.5990355610847473, 0.9292892217636108, 0.6963698863983154, 0.5983884334564209, -1.687095046043396, 0.28139397501945496, 3.2429370880126953, 0.40633678436279297, 1.0278632640838623, 0.016515687108039856, -0.3483928143978119, 1.6429691314697266, -0.8960202932357788, 0.05059155076742172, 0.5601009130477905, 1.7257311344146729, -1.079398512840271, -0.4978230595588684, -0.5584450960159302, 0.6775603294372559, -0.23595720529556274, 0.10722973942756653, -0.6683835387229919, -0.3445376455783844, -0.5651339292526245, 1.765061378479004, -0.9939673542976379, 1.5472526550292969, 1.0010205507278442, -1.8707317113876343, -0.45968687534332275, -1.3284567594528198, -0.4347168207168579, 0.7009892463684082, -1.5530248880386353, 0.917467474937439, 0.7381969690322876, 0.6799136400222778, 0.644628643989563, 0.3434235453605652, -2.6947813034057617, -1.5450434684753418, -0.09571444988250732, -0.12710654735565186, -0.6696391105651855, -1.263288974761963, -0.7183654308319092, 2.5338969230651855, -0.15422533452510834, 0.27629101276397705, -0.7988157868385315, 0.6651476621627808, -1.912287712097168, -2.146782159805298, -1.329460859298706, -1.701704502105713, 0.6328862905502319, -2.805325746536255, 0.20035028457641602, -1.6950926780700684, 1.1950501203536987, 1.3277723789215088, 0.17586955428123474, 1.1084421873092651, 1.3252742290496826, 2.5532050132751465, 0.36701202392578125, -0.7467045783996582, -3.8514750003814697, -0.6089208126068115, -0.8912861347198486, -0.5439869165420532, 1.543738603591919, 0.02212931215763092, 1.699310541152954, 0.2460205852985382, 0.2791041135787964, -0.576943039894104, -1.7667689323425293, 0.10939300060272217, 1.1703342199325562, -0.3821171522140503, -0.1704019010066986, 0.12891390919685364, -2.2120752334594727, -1.612736701965332, 0.7299192547798157, 1.2304998636245728, -1.9675767421722412, 0.3634587526321411, -0.3364598751068115, -0.0872964859008789, -0.3937731683254242, -0.0683518648147583, -0.018443360924720764, 0.558352530002594, 0.5362542867660522, 0.9674909114837646, 1.1227420568466187, -1.3174686431884766, 0.175568625330925, -0.6407759189605713, 1.197115182876587, -0.6437284350395203, -2.374021053314209, 0.3023150861263275, -1.486731767654419, 2.4411473274230957, 2.147749185562134, 0.745506763458252, -1.6401681900024414, -1.3749415874481201, -1.168869137763977, -1.3581737279891968, -4.744400978088379, 0.0280170738697052, 0.4592146873474121, 0.7357622981071472, 0.8576083183288574, -1.8746960163116455, -0.49700868129730225, 0.16675439476966858, -1.3056232929229736, -1.3066350221633911, -2.2519454956054688, 0.5308577418327332, 1.4904696941375732, -0.7625412344932556, 0.8329811096191406, -1.643500566482544, 1.0784876346588135, 2.0931997299194336, -0.05978727340698242, 0.4646734595298767, 0.9470625519752502, 1.253047227859497, 0.2784148156642914, -0.01111498475074768, -0.7269266843795776, 1.7549846172332764, 1.2616115808486938, -1.296046495437622, -2.4462509155273438, -1.1012563705444336, 0.20404493808746338, -0.12167687714099884, 0.5567527413368225, 1.8378190994262695, 2.3292994499206543, -2.110544204711914, 0.26478224992752075, 1.693660855293274, 0.060196876525878906, -1.4121410846710205, -0.016729027032852173, 0.5134304761886597, 0.8935190439224243, 0.02878570556640625, -2.606613874435425, 0.31348085403442383, 1.8393700122833252, -0.7375439405441284, -1.5546271800994873, -1.3142991065979004, 0.462080717086792, 0.38260984420776367, -2.3385438919067383, -0.43635475635528564, -0.1815078854560852, -0.3851965069770813, 0.6193916201591492, -2.2333059310913086, 1.5110139846801758, -0.35269808769226074, -0.6492782235145569, -1.5996325016021729, -3.3301753997802734, 2.038132429122925, -0.8659391403198242, 1.140913963317871, -0.9187476634979248, -1.4105883836746216, -3.022854804992676, 1.9639441967010498, -0.7768682241439819, -0.012923717498779297, -0.08293229341506958, -3.1522293090820312, 1.5875598192214966, -0.4934401512145996, 4.226912498474121, -0.6663461923599243, -0.684043288230896, -0.3557552695274353, -0.2754133939743042, -1.4122567176818848, 1.8712517023086548, -0.3798529803752899, -2.1575050354003906, 0.1765015721321106, 0.7499192953109741, 0.16821718215942383, 0.6113216876983643, -2.1765248775482178, 0.5786383748054504, 1.4553340673446655, 0.27698007225990295, -1.283287763595581, -0.1704624891281128, -1.6279387474060059, 1.0100560188293457, -0.9408520460128784, 0.875853955745697, -0.5124099254608154, -1.3120980262756348, -0.8901044130325317, 0.22521498799324036, 2.1658098697662354, -2.1326653957366943, -1.4486523866653442, -1.0368778705596924, 1.6684612035751343, -1.428741455078125, 0.03162179887294769, -0.0011966526508331299, 1.8069524765014648, 0.3953149616718292, -0.5286403894424438, -1.7841408252716064, -1.243116855621338, -1.6455578804016113, -1.5425009727478027, -0.016785725951194763, -0.32546067237854004, 1.0620614290237427, 0.5082732439041138, 0.9271500706672668, 1.6072865724563599, -1.0581378936767578, 1.5113639831542969, -1.9588533639907837, -1.1054264307022095, -2.0790786743164062, -1.7565629482269287, -0.8947250247001648, 0.307847261428833, -0.9129853248596191, 0.1556922197341919, -0.5227830410003662, 0.9079944491386414, 0.6834877729415894, 2.767636299133301, 1.3253380060195923, 2.1224918365478516, -1.2767752408981323, 1.9335647821426392, -0.45580387115478516, 0.4030507802963257, -1.669830560684204, -1.0866951942443848, -1.2471046447753906, 0.8498497605323792, 2.854593515396118, 0.8240891695022583, 0.5335395932197571, -2.4506783485412598, 0.054134488105773926, 0.5464272499084473, -0.09258443117141724, -2.5366921424865723, -1.5782698392868042, -1.4782887697219849, -1.6121153831481934, -1.0331029891967773, -0.28442859649658203, -0.8139734268188477, 1.1734884977340698, -0.9492666721343994, 0.7575325965881348, 0.16649949550628662, -0.5960441827774048, -0.5207901000976562, -2.482741594314575, -0.9493690729141235, 2.1800267696380615, 0.21016216278076172, 1.1145801544189453, -1.0176671743392944, -0.6757744550704956, 0.9832730293273926, -1.0651636123657227, -1.4416481256484985, -1.4921506643295288, 2.6464059352874756, 2.070207118988037, -2.192802906036377, 2.548935890197754, 0.7098099589347839, -0.7020970582962036, -0.15140286087989807, 0.9684620499610901, -0.17810654640197754, 0.7131142616271973, 3.2047555446624756, 0.2688852846622467, 1.9200224876403809, 1.4323983192443848, -1.4862139225006104, -1.1278916597366333, -0.15004697442054749, -0.21613958477973938, -0.5718519687652588, -0.9704008102416992, -1.2375102043151855, -2.150332450866699, 0.7119563817977905, 1.5518262386322021, -2.459469795227051, 2.1132144927978516, 1.315304160118103, 1.0186916589736938, -2.4672932624816895, -0.630186915397644, -0.03759963810443878, -0.42272329330444336, 0.8654109239578247, 0.45493388175964355, -0.35638755559921265, 2.6451430320739746, -0.7779028415679932, 0.9362027049064636, 0.6917644143104553, 1.5051805973052979, 0.18920326232910156, -0.11678504943847656, -2.99714994430542, -0.24379289150238037, 1.5588412284851074, 0.3640199899673462, 2.1567325592041016, 0.7299396395683289, -2.755295753479004, 0.3216570019721985, -1.2283469438552856, 1.6404054164886475, -1.350113034248352, -0.7898683547973633, -2.0433120727539062, -0.4484279155731201, -0.18936580419540405, -1.429026484489441, 3.07060170173645, 1.0435690879821777, -1.6574112176895142, 1.5792253017425537, -1.5002543926239014, 0.5700359344482422, -0.5954282283782959, 0.3457643985748291, 0.5966533422470093, 0.2930912375450134, 0.5104925632476807, -2.0231566429138184, 1.9279884099960327, -1.1309175491333008, 0.47522521018981934, -2.4158401489257812, -0.838756799697876, -0.373596727848053, -1.8997503519058228, 0.23156028985977173, -1.8781553506851196, -0.48820918798446655, 1.5989269018173218, 1.0437172651290894, -0.7181399464607239, 2.2297065258026123, 0.992267370223999, -1.733499526977539, -0.10922348499298096, 2.1549623012542725, 0.5000355839729309, 0.06884920597076416, 1.0779227018356323, 1.0764061212539673, 1.5721657276153564, 0.1141352653503418, 0.007246851921081543, 0.3649768829345703, 0.5288411378860474, -1.0202878713607788, -0.4359726309776306, -0.2639960050582886, -1.2913007736206055, 1.1671940088272095, -1.1015043258666992, -0.6713234186172485, 1.6893880367279053, -1.7222630977630615, -1.3614181280136108, 1.142472267150879, -1.141688585281372, -0.37375563383102417, -2.7158942222595215, 1.9479150772094727, 2.40779709815979, -4.390135765075684, 0.3941812515258789, -2.187433958053589, 1.1004328727722168, -1.1352267265319824, 1.8340147733688354, -0.16119009256362915, -0.009751200675964355, -0.8731601238250732, 2.2473340034484863, 3.0810368061065674, -1.804654836654663, -1.2996289730072021, -2.264697790145874, -1.8146376609802246, -2.160975933074951, 0.9561194181442261, 1.9584157466888428, -0.5984532237052917, 1.2287155389785767, -0.6522454023361206, -0.19796299934387207, -0.831382155418396, -2.2468600273132324, 0.36197125911712646, -2.76851749420166, -0.10704918205738068, -0.8028268814086914, -0.07966944575309753, -1.4658161401748657, -0.37518298625946045, -0.3059810996055603, -2.8485963344573975, 1.4447740316390991, -1.6182301044464111, 0.8605504631996155, -0.3813667297363281, 1.6955405473709106, 0.04131674766540527, -0.41394129395484924, -2.2008748054504395, 1.5180376768112183, -2.3974971771240234, -1.1672030687332153, -0.7888051867485046, -0.23785614967346191, -1.032351016998291, -1.582058072090149, -0.0026818513870239258, 2.1016035079956055, -0.3564918637275696, -0.03261488676071167, -1.6607706546783447, 0.9405262470245361, -0.568962574005127, 2.3971776962280273, -1.4842512607574463, -2.096348762512207, -1.7625539302825928, 2.0483763217926025, 1.5298972129821777, 0.8062785863876343, -1.3873400688171387, 1.9586567878723145, 0.1540539264678955, 2.126187324523926, -0.8554084897041321, -1.063833236694336, -0.9831606149673462, 1.1415812969207764, -0.6296491026878357, 0.6617835164070129, -0.7370033264160156, -1.800227403640747, 0.30924224853515625, 0.9650575518608093, 1.2082570791244507, -0.4299584627151489, -0.3283880949020386, 1.1101794242858887, 1.6500468254089355, -0.09618057310581207, 0.4751766324043274, 1.5593791007995605, -0.7412928938865662, 0.5848485231399536, 0.3851301670074463, -2.43220591545105, 0.5425711870193481, 0.3171966075897217, -1.8299931287765503, -1.1304433345794678, -1.1414802074432373, 0.14671671390533447, 1.6860188245773315, 1.8236949443817139, 1.068387746810913, -1.4783101081848145, 0.30016303062438965, -2.1663694381713867, -0.9249573945999146, -0.22216850519180298, 0.33645695447921753, 1.8741861581802368, -0.7903684377670288, -1.28999924659729, 0.09456631541252136, 2.2064876556396484, -0.2438054382801056, 1.8822987079620361, -1.2056316137313843, 0.4998387098312378, -1.4090114831924438, -0.08832317590713501, -1.6917662620544434, -1.821254849433899, 1.0900449752807617, 2.2731878757476807, 0.11139321327209473, -1.0806559324264526, -2.448788642883301, -0.8828566670417786, -0.6977308988571167, -3.391903877258301, 4.737201690673828, -0.7640925645828247, 0.6911282539367676, 1.9829654693603516, 1.596467137336731, 1.2156256437301636, -1.3030190467834473, -1.9604613780975342, -0.8637882471084595, -0.8420377373695374, 1.6527587175369263, 1.2661924362182617, 1.5558804273605347, 0.8052020072937012, -1.1346546411514282, -1.1336910724639893, 4.745467185974121, 0.4102201759815216, 0.8527594804763794, -0.009913861751556396, -1.8087661266326904, 0.7169249653816223, 0.14542511105537415, 0.38379180431365967, 1.172128438949585, -2.103044271469116, 1.1928236484527588, 2.664422035217285, -0.3586452901363373, 1.743552565574646, 1.2749030590057373, 0.10154161602258682, -1.8017053604125977, -0.09348905086517334, -1.2939422130584717, 0.6101349592208862, 0.3980875015258789, 1.521001935005188, -0.5756767988204956, 0.8461720943450928, -0.7770595550537109, -0.2883606553077698, 1.4636694192886353, -1.0537110567092896, -0.3663262128829956, -0.5227878093719482, -2.5957179069519043, -0.49600040912628174, 1.264953374862671, -2.446471929550171, -1.1907418966293335, 1.3738638162612915, 1.5307129621505737, -0.4830613136291504, -0.4726599454879761, 1.3356071710586548, 0.3615841269493103, -1.3510233163833618, 0.8805597424507141, 2.432255983352661, -0.11898083984851837, 0.6282621026039124, 0.05303185433149338, -0.37097296118736267, 0.21738819777965546, 0.3756840229034424, -0.7884664535522461, 1.0320104360580444, 0.4053921401500702, 0.693140983581543, 1.6357638835906982, -1.2802058458328247, 0.7767971754074097, 1.0801200866699219, -0.7986060380935669, -0.4070972204208374, 1.2220944166183472, -1.2771670818328857, 2.292991876602173, 2.4506912231445312, -1.4323806762695312, -0.5132391452789307, 0.43675968050956726, 1.8307384252548218, -2.3068904876708984, -0.574916422367096, -0.1751793920993805, -1.9940931797027588, 0.4200865626335144, -3.7897307872772217, 1.5795667171478271, 0.6035430431365967, 0.7750712633132935, -0.6696760654449463, -0.4466851055622101, 1.2203980684280396, 0.6097276210784912, 1.8042259216308594, 0.8673306107521057, -2.2470858097076416, -0.11085409671068192, 1.6956915855407715, 1.4746485948562622, 3.119035243988037, 2.316669464111328, -1.1500188112258911, -0.6047642827033997, -1.275214672088623, 4.9438300132751465, -2.0893099308013916, -0.39000698924064636, 2.4620728492736816, 2.107849597930908, 1.7925498485565186, -1.326021432876587, 0.33414459228515625, -0.501714825630188, 3.3702595233917236, -0.8509083986282349, -1.168299674987793, -0.05058497190475464, 0.5020259022712708, 0.766245424747467, -2.4079060554504395, -0.8443185091018677, -1.7272412776947021, 1.077478289604187, 0.07830122113227844, 0.2874247133731842, -0.3414485454559326, -1.670363187789917, 0.03368160128593445, -0.8612772822380066, -0.7335524559020996, 1.302025318145752, 1.0814697742462158, -0.717150092124939, -0.05906134843826294, -1.9459631443023682, -2.059518575668335, 1.5118560791015625, -2.3314216136932373, 1.4872095584869385, -1.2682373523712158, -0.7281703948974609, -0.9203788638114929, -1.8787106275558472, -0.0356670618057251, 0.7979788780212402, 0.9655206203460693, -1.0593090057373047, 0.751453697681427, 0.7295162677764893, 0.40467414259910583, 0.02390614151954651, -1.1611173152923584, 1.8793525695800781, -0.1654013693332672, 0.4027395248413086, 0.49010512232780457, -1.466004729270935, 0.5062880516052246, 1.1686034202575684, -1.9725947380065918, -0.9402161836624146, 0.8854855895042419, -0.43929600715637207, -2.0941755771636963, -0.23989789187908173, 1.982833743095398, -0.19464623928070068, 0.5065481662750244, 1.5705960988998413, 0.15731650590896606, 0.3306717872619629, -0.5325129628181458, -0.5042130351066589, -1.5711441040039062, -0.46664392948150635, 0.19816511869430542, -0.693973183631897, 1.698377251625061, 0.29719290137290955, -1.5403807163238525, -0.5152192711830139, 0.4347054362297058, 1.218265414237976, 0.9328460693359375, 3.0155458450317383, 1.6820201873779297, 1.5185539722442627, -1.296394944190979, -2.072221517562866, 0.7130190134048462, 1.596859335899353, -1.731090784072876, -0.03214132785797119, 3.225736141204834, -0.9426648020744324, -0.30435657501220703, -1.1085313558578491, -0.023798584938049316, 1.5415129661560059, -0.4672248065471649, -0.3256869316101074, 1.0971899032592773, -1.9043869972229004, 2.6090917587280273, -1.0256519317626953, 2.971968173980713, 0.7227169275283813, 0.5253034830093384, -0.8210798501968384, 1.5803253650665283, 0.3366113007068634, -2.2404956817626953, 1.455047845840454, -1.2678391933441162, -1.1953105926513672, 0.0021062493324279785, 1.1121959686279297, -1.1919304132461548, 0.41271647810935974, -1.2496402263641357, 1.1930582523345947, 0.1785922646522522, 2.132734775543213, -1.602433681488037, 2.545715093612671, 0.7524297833442688, -1.6973755359649658, -0.1057593822479248, 1.0002961158752441, -1.0519484281539917, 0.7338775396347046, 2.8222177028656006, 2.0383682250976562, 0.2411298155784607, 1.5025502443313599, 2.69189715385437, 0.7103844881057739, 0.2814592123031616, 1.5648810863494873, 0.2999965250492096, 1.7501332759857178, 1.6686747074127197, -1.4422943592071533, 1.5214861631393433, -0.04607999324798584, -2.642150640487671, 0.4464818239212036, -0.5938825607299805, 0.1312437653541565, 0.5905797481536865, -0.17681491374969482, 0.2737407386302948, 0.1874173879623413, 2.3909125328063965, 1.1525051593780518, 0.9430753588676453, -0.9526029825210571, 1.2122751474380493, -2.844320297241211, 0.8750953674316406, -1.0867345333099365, -1.7722654342651367, 1.167327642440796, 2.0011048316955566, -1.8566744327545166, 1.3836746215820312, -1.0506620407104492, -1.1451096534729004, 0.9310283660888672, -0.9993875026702881, 0.8704967498779297, 0.46838241815567017, -0.41571396589279175, 2.114462375640869, -1.055105447769165, 1.3919496536254883, -2.3423454761505127, 3.5426828861236572, -1.168121337890625, 1.2442805767059326, -0.3121964931488037, 1.1154850721359253, 1.2891744375228882, 0.2930263578891754, 0.12708261609077454, -0.1589905023574829, 0.42723703384399414, 0.4149780869483948, 0.18267950415611267, 0.43189939856529236, -1.3001534938812256, -0.16724443435668945, 0.03193576633930206, -0.061102867126464844, -0.9630380868911743, 0.4990133047103882, 2.2423505783081055, 0.47915521264076233, 0.9102624654769897, -0.010438352823257446, -1.4698654413223267, 1.288230299949646, 0.039674192667007446, -2.9261860847473145, -2.200680732727051, 0.994255006313324, 1.1207988262176514, 2.0166845321655273, -0.12423647940158844, 2.0761632919311523, -0.4616226255893707, 0.420165479183197, 1.9213577508926392, 0.4314653277397156, 0.08803118020296097, 1.933878779411316, -0.5483011603355408, -1.8053393363952637, -1.4973125457763672, 0.6141946911811829, -0.41402095556259155, 1.3706042766571045, 0.6914201974868774, -1.0365608930587769, -0.619715690612793, -2.6117780208587646, 1.480406403541565, 0.5306580662727356, -1.121612787246704, -0.20350605249404907, 0.40613412857055664, -0.8971344232559204, 0.2647203207015991, -1.2113990783691406, -0.5560898780822754, 0.8761489391326904, -0.30110833048820496, -0.3618086874485016, 0.3164077699184418, -1.9698011875152588, 2.2141189575195312, -0.9611133337020874, 0.36876073479652405, -1.6278493404388428, -0.907856822013855, 1.030632495880127, 0.7414742112159729, 1.7102652788162231, 2.126228094100952, 1.9332432746887207, 1.3721725940704346, -1.295164704322815, -0.980785608291626, 1.9405107498168945, 0.5223232507705688, -0.6155006885528564, -1.360225796699524, 1.0394856929779053, -1.447470784187317, -0.31658193469047546, -0.5565435290336609, -0.8822554349899292, 0.41540956497192383, -1.495632290840149, -0.8937293291091919, -1.7009811401367188, -0.8538014888763428, -1.258622646331787, -0.17254900932312012, 0.5647113919258118, 0.9835294485092163, -0.7732880115509033, 0.902015209197998, 0.17349626123905182, -1.8825780153274536, 2.3274171352386475, -0.3819906413555145, 0.7001609206199646, -2.6544723510742188, -0.18504051864147186, -1.3483467102050781, -0.16941821575164795, 1.3044158220291138, -1.39686119556427, 2.526718854904175, -1.2474132776260376, -0.18188029527664185, 0.7198725342750549, -1.619692325592041, 0.634798526763916, -1.6908576488494873, -2.11030912399292, 1.3243911266326904, -1.9511041641235352, 2.2809011936187744, -0.2835811674594879, 1.466810703277588, 1.132082462310791, -0.2076929360628128, -0.08126139640808105, 1.5876001119613647, 3.1880431175231934, 1.8231021165847778, -1.982168197631836, 0.7262697815895081, 2.973893165588379, 0.2937236428260803, 0.847577691078186, -1.7528750896453857, -0.88761305809021, 0.06094425916671753, 0.11068093776702881, 0.5868258476257324, 0.88492351770401, 1.473865032196045, -1.8118559122085571, 1.9431434869766235, 0.26953089237213135, 0.09709107875823975, -1.9973636865615845, -1.190330982208252, -2.4318783283233643, -0.22720122337341309, -0.5921460390090942, 0.42907223105430603, -2.514631509780884, -1.3826324939727783, 0.12255747616291046, 3.07853364944458, -0.7232749462127686, -0.6756508350372314, -0.6444587111473083, -2.6900253295898438, -1.4502699375152588, 0.3830346465110779, 0.16998688876628876, 0.7296943068504333, -2.7663114070892334, 0.6237989664077759, -1.9624766111373901, 1.4091591835021973, 0.9371405243873596, -3.207679271697998, 2.3273613452911377, -0.15806955099105835, -4.869596481323242, 1.8358681201934814, 0.05687671899795532, 1.780472755432129, -0.6894278526306152, -1.491391658782959, 0.03172659873962402, 0.6370751261711121, 1.854553461074829, -1.324586272239685, 2.5042781829833984, 0.3196885287761688, -0.9137916564941406, 0.9352675676345825, -1.2151224613189697, 0.7447825074195862, -1.1277556419372559, -1.6405465602874756, 3.0199179649353027, 0.3684287667274475, -1.463574767112732, -1.6108657121658325, 0.7445234060287476, 0.5209019780158997, -0.06025010347366333, 0.6437490582466125, -0.1931363344192505, -0.9822500348091125, 0.154746413230896, -0.3627321124076843, -1.2038590908050537, -1.6670045852661133, -2.5746612548828125, 2.8974361419677734, -0.18758687376976013, 1.0822248458862305, -0.14337468147277832, -0.7768986821174622, -1.9645075798034668, -0.48088183999061584, -2.318225860595703, 2.6630566120147705, 2.26728892326355, 0.571617066860199, 0.7047873139381409, 0.44036900997161865, -0.44619935750961304, 0.7456589341163635, -1.9468820095062256, -0.571054220199585, -1.8941168785095215, -0.14448755979537964, 0.26247596740722656, 0.1080518364906311, -0.41023245453834534, -0.23200613260269165, -0.040071845054626465, -0.5179673433303833, -1.759714126586914, 0.3911323547363281, -3.4932570457458496, -0.36990711092948914, -0.48988306522369385, 0.9931468963623047, 0.32122373580932617, 0.04196375608444214, -2.151841402053833, 1.354348063468933, 0.6709578037261963, -0.34452545642852783, -0.4891103208065033, 2.98829984664917, -0.005372226238250732, -1.4630112648010254, 2.036205291748047, -0.2716563940048218, 0.7463371753692627, 0.43389034271240234, 0.7279173135757446, -1.3387184143066406, -0.4849509596824646, 0.22430962324142456, -1.1431466341018677, 0.3309340178966522, -0.08386394381523132, 0.5339478254318237, -1.193253517150879, 2.145253896713257, 1.0294623374938965, -1.1104328632354736, -5.0669145584106445, -1.0181382894515991, 1.1090362071990967, -1.25894296169281, -1.5731528997421265, 1.5832384824752808, 1.613759994506836, -0.23480352759361267, 1.1223400831222534, -0.6978374123573303, -1.8269513845443726, -0.17617759108543396, 0.43476447463035583, 0.8808630108833313], [0.18593204021453857, 1.7313742637634277, 1.648781180381775, 0.16363286972045898, -0.7337496876716614, 1.1732759475708008, -1.753245234489441, 0.9681617021560669, -2.5578081607818604, 1.4963948726654053, 1.1895802021026611, -0.2067309021949768, 1.5478603839874268, -1.677190899848938, -0.4267657995223999, 0.09340023994445801, -0.5276542901992798, -0.5190516710281372, 1.4185960292816162, -0.9088140726089478, -0.3629782795906067, -0.33540263772010803, -0.969537079334259, 0.38076460361480713, 0.342201828956604, 0.7486388683319092, -0.36510708928108215, 1.535452961921692, -1.3028813600540161, -0.9045263528823853, -0.8166319131851196, 0.8792955875396729, -0.7126514911651611, -0.6571400165557861, -1.1583303213119507, -0.08066907525062561, -1.868493914604187, -1.4722297191619873, 0.6494404673576355, 3.245521068572998, -0.5353518724441528, 0.437500536441803, -1.0097522735595703, -0.608464777469635, 2.091366767883301, 2.0709166526794434, 0.1055377721786499, -1.8785555362701416, -1.3775205612182617, 0.8233753442764282, 1.4507077932357788, 0.18651962280273438, 1.921478271484375, 0.8025285601615906, -0.46175616979599, -2.02740740776062, -1.2304587364196777, 0.06477940082550049, -0.5701850652694702, 1.1437910795211792, 0.8107842206954956, -0.7103066444396973, -0.8561968803405762, -0.43119657039642334, -1.2564281225204468, 0.44392600655555725, 0.07801929116249084, 4.740675926208496, -0.3816957473754883, -0.5578705072402954, 1.8862351179122925, -0.06981980800628662, -3.656203269958496, 1.5629345178604126, -1.6575438976287842, 0.15798485279083252, -1.6295244693756104, -1.6450272798538208, 0.20283007621765137, 3.521613359451294, 1.1672401428222656, 1.3472900390625, 0.20862460136413574, -0.19057440757751465, -0.7436308860778809, -0.4701899588108063, 0.6354734897613525, 0.5930918455123901, -0.1379835307598114, -0.8456670641899109, -1.168641209602356, -0.25085026025772095, 1.5605090856552124, -0.21452432870864868, -1.1072826385498047, -0.8001850843429565, 0.30595213174819946, -1.0507137775421143, -0.861769437789917, -0.1170264482498169, 0.3984391391277313, -0.7595239877700806, -0.9473503828048706, 1.2594726085662842, 1.3667138814926147, -1.943727731704712, -1.1627329587936401, -1.9681916236877441, -0.6023870706558228, -0.4798433780670166, -0.09303862601518631, 2.599886178970337, 1.6920145750045776, 1.1247222423553467, -1.6500861644744873, 0.5726380348205566, -2.1297669410705566, -1.3752412796020508, -0.027047276496887207, -2.527627468109131, 0.3396901488304138, 2.976651191711426, -0.08898119628429413, -0.48542320728302, -1.8655985593795776, 0.48092320561408997, 0.6737784147262573, -1.0863099098205566, 0.03865480422973633, 2.2351152896881104, -0.03316044807434082, 0.23507729172706604, 1.8999940156936646, -2.4545304775238037, 0.651416540145874, 0.8818339109420776, -1.9212751388549805, 0.4067612290382385, -0.547970712184906, -1.8438187837600708, 0.18311870098114014, -0.2662075161933899, -1.5960185527801514, -0.8653569221496582, -3.7563910484313965, -1.180085301399231, 1.0670685768127441, 1.0075130462646484, 2.298021078109741, 0.3281348943710327, 0.9114251136779785, -0.8834888339042664, -2.735710382461548, -0.277798056602478, 2.8220882415771484, 1.3462815284729004, 1.4767887592315674, 0.2770591378211975, -0.4361109137535095, 1.1707426309585571, 0.7490496635437012, 0.6152433156967163, 0.7373586297035217, 0.3970770239830017, -2.747169256210327, -0.6112138628959656, -1.1484912633895874, 1.0424798727035522, 3.2112913131713867, 2.7554893493652344, -1.5409860610961914, 1.3275431394577026, -1.3872789144515991, 1.6023715734481812, -1.297905445098877, 0.06494200229644775, -1.1512917280197144, -0.2291099727153778, -2.018315076828003, 1.374538779258728, -0.5403586626052856, 0.9330804347991943, -1.1714953184127808, 0.9981412887573242, 0.22008579969406128, -0.9008201360702515, -1.2321937084197998, -1.2216213941574097, 0.17776934802532196, 0.4636431932449341, 1.342037558555603, -0.43971872329711914, -0.5427451729774475, -0.26497185230255127, -0.9633335471153259, 1.4404540061950684, -0.9342976212501526, 0.993137001991272, -0.6720983982086182, 1.778451919555664, -1.6358668804168701, -0.3834320902824402, 0.18537092208862305, -1.204056739807129, 1.5742266178131104, -0.5955835580825806, 1.9755399227142334, -0.9815449714660645, -3.3009610176086426, 1.775130033493042, -3.5024173259735107, -0.27207010984420776, -0.16846206784248352, 0.5564793348312378, 2.309215784072876, -1.4593234062194824, -0.8430873155593872, -2.844674825668335, 2.0048623085021973, -0.8053827285766602, -0.8143740892410278, 1.7335363626480103, -0.8403196334838867, -0.5652629733085632, 1.145225167274475, 3.3986542224884033, -0.57865309715271, -0.8757695555686951, -3.8775124549865723, -1.0204026699066162, 0.8237591981887817, -0.5526492595672607, 0.054270874708890915, -0.5692936778068542, 0.08107167482376099, 1.6685504913330078, 1.1065391302108765, -0.39968764781951904, 0.9436643123626709, 0.10158586502075195, -0.07259058952331543, 0.7835643291473389, -0.7966979742050171, -1.878901481628418, 1.3483762741088867, -0.6022862792015076, -0.24088159203529358, 0.6790121793746948, -0.9347569942474365, -1.0227423906326294, -0.09457051753997803, -0.5873634815216064, -3.049268960952759, 1.894420862197876, 0.9370473623275757, -2.121903419494629, 2.1440906524658203, 1.0412659645080566, -1.1287213563919067, -0.13702011108398438, -1.1397500038146973, -1.7841283082962036, -0.900325357913971, -1.7348618507385254, 1.3788954019546509, 0.11000746488571167, -1.5715795755386353, 0.4465380609035492, -0.19115042686462402, -2.43979549407959, -0.7711185812950134, -0.3603872060775757, -1.23553466796875, 0.9488580226898193, 1.2916486263275146, 1.8789458274841309, 0.24876542389392853, -2.721592664718628, -1.7865930795669556, 0.3963083028793335, -1.5093319416046143, 3.456460475921631, 2.2764265537261963, -0.4462529122829437, -0.6272197961807251, -0.6802802085876465, 0.7776453495025635, -1.8037374019622803, -0.5586493611335754, 1.4383255243301392, 0.1078188419342041, 0.30344322323799133, -0.8073620796203613, 0.18808192014694214, 1.6760942935943604, 2.4264111518859863, -1.3737157583236694, -1.0420663356781006, 1.0592297315597534, 0.015397101640701294, 0.6888269186019897, 0.5403059124946594, 1.7676548957824707, -0.8924620747566223, 1.2633472681045532, -0.6987094283103943, 2.243363857269287, -0.1977589726448059, 0.9051187038421631, -0.462862104177475, -0.8858839869499207, 0.050690531730651855, -1.9610530138015747, 2.684037685394287, 1.495870590209961, -0.7914019823074341, 0.1545732021331787, 0.6956428289413452, -0.16472002863883972, 0.6355948448181152, -0.4254331588745117, 0.22176730632781982, 1.6897317171096802, 0.7044496536254883, 1.5237212181091309, -1.6840490102767944, -1.995629072189331, -0.2562446892261505, -0.5832794308662415, -0.5668203830718994, 0.38637498021125793, -0.9309555292129517, -0.744581401348114, 0.3633253872394562, 1.848480224609375, -0.6427600383758545, -0.2504853308200836, -1.4662773609161377, 0.14305657148361206, -1.174281120300293, -0.3812130391597748, 1.6622240543365479, 2.3079354763031006, 1.5412260293960571, -1.7066669464111328, -0.1170475035905838, -1.4064509868621826, 1.4623191356658936, 1.7117434740066528, -1.7406426668167114, 0.14620567858219147, -0.09382572770118713, -0.3640608787536621, 1.1071653366088867, -1.02908456325531, -1.625333547592163, 2.032663106918335, 1.3425339460372925, 0.8228088021278381, -0.42855894565582275, -2.0544679164886475, 1.5590416193008423, -1.840489149093628, -0.6730929613113403, 1.2516565322875977, 0.9921479821205139, 1.093145728111267, 1.150797724723816, -0.2539174258708954, -1.2681121826171875, -0.46838194131851196, 1.4933093786239624, -0.9798002243041992, 0.30594733357429504, -0.3682507276535034, -0.08859473466873169, 1.3842272758483887, -0.2585054039955139, -2.7484304904937744, 1.0157777070999146, -1.554797887802124, -1.9365911483764648, -2.6131107807159424, -1.372255802154541, -1.688796043395996, -1.109843373298645, -1.0373430252075195, -0.5677173137664795, -0.7395581603050232, 0.4399557113647461, -1.8463771343231201, 1.9042596817016602, -2.3193717002868652, -2.2036938667297363, 0.9678000211715698, 0.18268698453903198, -0.6294676065444946, 1.8773469924926758, 0.22710883617401123, -1.342118501663208, -0.06926307082176208, 1.9570562839508057, -0.35904252529144287, 0.46178752183914185, 1.1577218770980835, 0.029855787754058838, 1.1618971824645996, 0.6270202398300171, -1.6106164455413818, -2.184551477432251, 0.26502227783203125, 0.3374391496181488, 0.6129897236824036, -1.4452036619186401, -2.4050748348236084, 0.0804523229598999, 1.3515169620513916, -0.2073630690574646, -0.11825990676879883, 1.897294521331787, 0.39697352051734924, 0.10409712791442871, -1.0679556131362915, 0.47517824172973633, 1.014582872390747, -0.7315909266471863, -0.5783453583717346, 0.3771587312221527, 0.3331398665904999, -0.43390703201293945, -0.8638591766357422, -0.6060313582420349, 0.2691086530685425, 1.0307347774505615, 2.7495718002319336, 0.02783644199371338, -0.8350089192390442, -0.07208632677793503, -1.0988099575042725, -0.029764369130134583, -0.7642000913619995, -2.283479690551758, -0.6653755903244019, 1.556875228881836, -1.2647181749343872, -1.76357102394104, -1.235323429107666, -1.196047067642212, -0.40458211302757263, 1.1978886127471924, -0.059786200523376465, 2.0166444778442383, -0.11460983753204346, 2.3458783626556396, 0.08173006772994995, -0.27814781665802, 0.3694160580635071, 0.4228912591934204, 0.49110472202301025, 0.3488137125968933, 1.3124608993530273, 2.7189745903015137, -1.7955656051635742, 0.6454998254776001, -1.5353130102157593, 1.0841546058654785, 1.9406788349151611, 1.2410550117492676, -1.4948644638061523, 2.365185499191284, -1.8392720222473145, 0.8241276144981384, 1.0365103483200073, 1.142994999885559, -0.7918330430984497, -0.7340131998062134, -0.8305328488349915, 1.4331042766571045, -1.6960452795028687, 0.6494839787483215, -1.4585039615631104, -0.25616157054901123, 0.5500130653381348, 0.9184251427650452, -0.0389133095741272, -1.1451776027679443, 0.7694424390792847, -2.6213388442993164, 1.3199443817138672, 0.154526025056839, -0.3583565652370453, -0.040944963693618774, 2.1060948371887207, 1.4703264236450195, 0.10288691520690918, 2.087124824523926, 1.7356597185134888, 3.905989170074463, 1.3290014266967773, -0.09612096101045609, 0.2628941535949707, 0.9836872220039368, -2.564471960067749, 0.7404419779777527, -1.6846120357513428, 1.0418856143951416, -2.214568614959717, -2.6044323444366455, 0.7709090709686279, 0.4078909754753113, -1.3066096305847168, 0.07637593150138855, -0.6167716979980469, -0.08005106449127197, -2.149812698364258, -2.474824905395508, 0.5489298105239868, 0.41744980216026306, 2.4957382678985596, -1.3438609838485718, -0.38088536262512207, 0.4814390540122986, 0.3620535731315613, 1.7773962020874023, -2.1034364700317383, -0.5571304559707642, -1.5023566484451294, -1.2352592945098877, -1.3602120876312256, -1.4893661737442017, 0.009141549468040466, -1.7583361864089966, -0.8181674480438232, -0.3705577254295349, -0.6217768788337708, 0.21943655610084534, -0.47785595059394836, 2.4850387573242188, 0.7606196999549866, 1.1759378910064697, 1.629669427871704, -0.3427761197090149, 0.2503364682197571, 2.133049964904785, 1.4717437028884888, -2.719085931777954, -1.4993622303009033, -0.6366790533065796, 1.4315259456634521, -0.7661293148994446, -1.6141301393508911, 1.1540493965148926, 1.9869437217712402, -1.920794129371643, -1.8437342643737793, 1.2802972793579102, 1.858184576034546, 1.5260703563690186, 1.0834884643554688, -2.2432518005371094, 0.6682199835777283, -0.7508243322372437, 0.020430266857147217, -1.4640083312988281, 1.2619316577911377, -0.9742553234100342, -2.4259002208709717, -1.7144218683242798, -0.8534823060035706, -1.562821626663208, 1.1825060844421387, 0.0093916654586792, -0.09011918306350708, -1.945162057876587, 1.6830235719680786, 1.6597850322723389, 1.9848737716674805, 2.7854223251342773, 0.7871071696281433, 2.1099789142608643, 1.5304865837097168, -2.1805386543273926, -1.1996941566467285, 0.39952558279037476, 1.1894001960754395, 0.30147361755371094, -0.972714900970459, -1.5095140933990479, 0.35386574268341064, -0.8687931299209595, -1.005372405052185, -0.24514645338058472, -0.6097484230995178, 0.009798884391784668, -1.0639451742172241, -2.1223440170288086, 1.1331756114959717, 1.2307446002960205, -1.0892953872680664, -1.423035740852356, 1.7262815237045288, 1.2441216707229614, 0.3472656309604645, 0.7758476734161377, -0.508765697479248, -2.911416530609131, -1.80328369140625, -1.5775409936904907, 1.3520530462265015, -0.3119714856147766, 1.2010462284088135, 0.053123027086257935, -2.4048914909362793, 1.1023608446121216, -2.205723285675049, -1.9625449180603027, -0.03413498401641846, -1.4800312519073486, -0.32043078541755676, 0.16302108764648438, 1.2965704202651978, -0.9175572395324707, 0.161911278963089, -1.0138177871704102, -0.3298991918563843, -2.5945281982421875, 2.025496482849121, 0.09740060567855835, -0.7831183671951294, 2.396836519241333, 1.4245609045028687, -2.3574838638305664, 1.1884191036224365, -1.3712472915649414, -0.8332045674324036, -1.1553951501846313, 1.4493948221206665, 1.9706735610961914, -0.3407379388809204, 2.4594202041625977, 1.333589792251587, 1.894545555114746, 0.5858505964279175, 0.5024535059928894, -3.2688589096069336, 1.8206443786621094, 2.761094093322754, -0.7475618124008179, -0.5003474950790405, 1.1327133178710938, 1.8419301509857178, -1.3928871154785156, 0.16051316261291504, -0.5520686507225037, -1.783051609992981, -1.397024154663086, 1.560093641281128, -0.8913453817367554, 1.1715476512908936, 0.0016688406467437744, -0.7209390997886658, 0.8394894003868103, -0.5600847005844116, 0.9472017288208008, 0.35447531938552856, 1.9430193901062012, -2.04410982131958, -0.31220394372940063, -1.2010574340820312, -1.708073616027832, -1.0465242862701416, -0.7178774476051331, 0.9580861926078796, -0.3529314696788788, -0.9364649653434753, 0.36169013381004333, 0.898937463760376, 1.5991665124893188, -0.06500568985939026, -0.029531776905059814, 1.5465788841247559, -1.071826696395874, 0.902636706829071, 1.236573338508606, 2.0852062702178955, 0.9733583927154541, 1.6930594444274902, 1.0749335289001465, 0.08603903651237488, 0.06539115309715271, 1.0471547842025757, 2.275965929031372, 0.48458635807037354, -0.111721932888031, 0.652708888053894, -1.5749256610870361, 1.9481233358383179, 0.1407732367515564, -0.22798335552215576, -2.185457944869995, 1.5782699584960938, 1.054678201675415, -0.560930073261261, -2.151962995529175, -0.21372705698013306, 1.3068861961364746, 0.9345462918281555, -0.5524230003356934, 3.189331531524658, -2.158039093017578, -1.0305575132369995, -2.252488613128662, 0.38725200295448303, -0.22309526801109314, 0.3332745432853699, 0.23476293683052063, 0.8219115734100342, -1.631223440170288, -1.5195575952529907, 1.0299971103668213, -0.7545427083969116, 0.40102261304855347, 1.1934888362884521, 0.2078981250524521, -0.4124765694141388, -2.231462001800537, 1.511894702911377, -1.3745137453079224, -0.8957175612449646, 0.6535905599594116, -1.7383568286895752, 1.6406924724578857, 1.1204755306243896, 1.2845256328582764, 0.43760591745376587, -2.297940731048584, -0.8259344696998596, 1.937670350074768, 1.045992374420166, 1.0193191766738892, 1.51877760887146, 2.348541021347046, -0.47003936767578125, 0.9346840977668762, -0.4446340799331665, -2.262843608856201, 1.0589474439620972, 1.5435967445373535, -0.8960535526275635, 0.2557050585746765, 0.8926416039466858, 0.05742475390434265, -1.5702972412109375, -0.0357930064201355, 0.9297308921813965, 2.123509407043457, 0.2550247311592102, 0.07760266959667206, -0.7456647157669067, -0.44647684693336487, 1.4811248779296875, 1.4405901432037354, -0.5065228939056396, -1.1682302951812744, 1.6693305969238281, 0.6986631751060486, -0.5563424825668335, 2.1091842651367188, 0.5718768835067749, -3.5377979278564453, 0.37157106399536133, 0.1411210298538208, 0.8318561315536499, 1.8803679943084717, -0.4152623414993286, 1.581648349761963, 0.5766878128051758, -2.347571611404419, 0.4652610421180725, -1.1237239837646484, -1.899851679801941, 1.7294820547103882, 0.1894603967666626, 1.3948602676391602, 0.2841588854789734, 1.4919993877410889, 0.1782280057668686, -3.068887948989868, 0.568260908126831, -0.5270880460739136, -0.07083821296691895, -1.2802661657333374, 2.392627716064453, -2.158745288848877, -0.6000792980194092, 0.6425195932388306, -0.5980739593505859, 1.0858659744262695, -0.2577872574329376, -1.1894545555114746, 1.1938409805297852, 2.9383158683776855, -2.270845413208008, 0.11275482177734375, 1.1278878450393677, 0.9254889488220215, 0.6767236590385437, -1.372983455657959, 1.3395068645477295, 2.8336949348449707, 4.0696120262146, 3.700272560119629, -1.1521986722946167, 0.44683825969696045, -2.691685914993286, 1.3500890731811523, 0.22269196808338165, -0.20276537537574768, 1.827113151550293, 2.001667022705078, 1.7739325761795044, 0.1469876766204834, 0.37553420662879944, -0.8879570960998535, 1.1222774982452393, -0.18721267580986023, -0.12174353003501892, -0.9680535197257996, 0.272666335105896, 1.3219884634017944, 2.338603973388672, 1.4357101917266846, 0.12335366010665894, -0.9733607769012451, 0.8523871898651123, -0.5950731039047241, -1.925438404083252, 0.4951257109642029, -0.5146162509918213, -0.4372349679470062, 0.7547252774238586, 1.0680046081542969, 0.3200908899307251, 0.2784048318862915, -0.13937127590179443, -1.0815565586090088, 0.12132835388183594, 2.001810312271118, -2.287548542022705, 0.25821352005004883, -0.9622893929481506, 1.259825587272644, -0.9279082417488098, -0.560974657535553, 2.0566813945770264, -0.7462037205696106, 0.015448331832885742, 2.275650978088379, 1.1791353225708008, -0.7827171087265015, -2.754337787628174, 0.6492406129837036, 0.3261754512786865, -0.611074686050415, -0.35941192507743835, -1.4686713218688965, -2.294182538986206, 0.539280116558075, -1.6441190242767334, 0.09325987100601196, 1.7921133041381836, 2.793145179748535, 1.5342133045196533, 0.06160271167755127, -2.0812180042266846, -1.1859500408172607, 1.0137633085250854, 0.30667173862457275, 0.43497419357299805, 1.1869832277297974, 1.5713406801223755, -0.6917256712913513, -0.6806305646896362, 0.8762814998626709, -1.8738023042678833, -1.4218072891235352, -0.3004594147205353, -2.6005680561065674, 0.5359511971473694, 0.8397245407104492, -0.06564411520957947, -2.9986095428466797, 1.0630079507827759, 0.5810465812683105, -2.1113383769989014, -1.9693117141723633, 1.1122045516967773, 2.7405834197998047, 0.6843267679214478, -0.44409114122390747, 0.37296974658966064, -0.0994613990187645, -2.1924428939819336, 1.4666438102722168, 0.7700227499008179, 0.32829493284225464, 0.9960296154022217, 3.1320791244506836, -0.45119747519493103, 2.4675917625427246, -0.16263851523399353, -0.6854735016822815, 1.2436530590057373, -1.4594281911849976, 0.8008015751838684, -0.772983729839325, 2.641993522644043, -0.3300873637199402, 0.9996505379676819, 3.1912357807159424, 1.1538081169128418, -1.3832085132598877, 0.48636510968208313, -1.6298482418060303, 1.3321253061294556, 0.3766274154186249, -1.759765863418579, 2.068793535232544, -1.026198148727417, 3.028254747390747, -1.9522602558135986, -0.4885200262069702, 0.709331214427948, -2.0317018032073975, -0.6730809211730957, 1.2984240055084229, 0.3118310272693634, 0.007623553276062012, -1.0900089740753174, 0.9797765612602234, -1.6924628019332886, -1.3980026245117188, 3.5873589515686035, 0.916576623916626, 2.0843148231506348, 1.6313802003860474, 0.836450457572937, 0.8863742351531982, 0.9022115468978882, -0.022620946168899536, -0.22886896133422852, -0.06695497035980225, -2.162337064743042, -0.5620209574699402, -2.4563727378845215, 1.2381647825241089, 2.1646580696105957, -0.11621612310409546, 0.6537889838218689, -0.7088240385055542, -0.7209117412567139, -0.638781726360321, -0.9896954298019409, -1.9484434127807617, 1.7230474948883057, -1.5934698581695557, 0.4087783396244049, -0.6094922423362732, 1.2916007041931152, 0.22560715675354004, -2.439070224761963, -0.6342493891716003, -1.569230556488037, 1.7258672714233398, 0.7261858582496643, -0.26432371139526367, 0.6046438217163086, -0.2960642874240875, 1.8313840627670288, -0.0053580403327941895, 0.8476546406745911, -0.31402480602264404, -4.1409149169921875, 1.6720056533813477, 0.09821507334709167, -0.24370121955871582, 1.1012191772460938, 3.890101194381714, -0.5730224847793579, 1.7201323509216309, -1.81038498878479, -1.791637897491455, 0.9573196172714233, -0.04286177456378937, -1.2113093137741089, 0.4024597406387329, 1.5442986488342285, -0.056512534618377686, -0.9516320824623108, -0.7822717428207397, -1.3919377326965332, 0.05097478628158569, -0.022948384284973145, 0.642227292060852, -0.9777076244354248, -0.9960298538208008, 0.5320680141448975, 2.2385921478271484, -0.07883447408676147, -0.09653714299201965, -0.32856786251068115, -1.330361008644104, 0.6301801204681396, 0.8039625287055969, -4.26940393447876, 1.1955437660217285, -0.9671165347099304, 0.7131017446517944, 0.10559773445129395, 0.3151598274707794, -1.5535365343093872, 2.8283138275146484, -0.9347827434539795, 0.37589389085769653, -0.14608798921108246, 0.4460718035697937, -2.36059832572937, 2.0883307456970215, 1.6614456176757812, 2.2698285579681396, 3.235374927520752, 1.8558387756347656, 1.2019071578979492, -0.32796698808670044, -0.4571119546890259, 0.7453714609146118, -0.9054215550422668, 1.0127021074295044, 2.117849349975586, -2.953181743621826, 1.2229007482528687, -1.3072524070739746, -0.09202778339385986, -2.679248332977295, 0.6915687918663025, 1.6852922439575195, 0.6265280842781067, 1.9079636335372925, 0.897716760635376, 0.5732325315475464, 0.5789807438850403, 0.4696256220340729, -0.16227887570858002, -1.5840919017791748, -0.0879870057106018, 2.6517815589904785, -1.2252088785171509, 0.6020705103874207, 2.3687901496887207, 0.3030371367931366, 0.6838202476501465, 0.9386156797409058, -1.041048288345337, -1.9321153163909912, 0.14025843143463135, 0.6672460436820984, 0.22749179601669312, -1.1150919198989868, 2.900230884552002, -2.053541898727417, 0.212571382522583, 1.59601891040802, 1.212514877319336, -0.2673983573913574, 2.17547345161438, -1.6374430656433105, 0.06946449726819992, -1.6701347827911377, -1.3218656778335571, -1.5744441747665405, -2.0090832710266113, -1.1053967475891113, 0.49312543869018555, 1.7236237525939941, -0.34575355052948, 0.972416877746582, -1.3726165294647217, 1.4652152061462402, 3.602571487426758, -2.795354127883911, 2.019603729248047, -0.4884931743144989, 0.2933320999145508, 0.24316871166229248, 0.23790530860424042, 1.5472042560577393, 1.84030020236969, -0.6853556632995605, 1.5705146789550781, 0.45879024267196655, -1.4175148010253906, -0.12106883525848389, 1.355135202407837, -1.5214874744415283, -1.781333565711975, 2.1700832843780518, -0.7705270648002625, -0.7210943102836609, 0.7062636613845825, -0.7587862610816956, -1.805213212966919, 1.026267409324646, 4.062575340270996, 0.2301294207572937, -1.5159428119659424, -2.8653564453125, 0.0038646459579467773, -0.853269100189209, 1.295400619506836, -1.728628158569336, -3.863055467605591, 1.8918793201446533, 0.8217471837997437, -0.6873888969421387, 2.34578275680542, 0.5905095934867859, -0.01315772533416748, 0.8745253682136536, 3.1845080852508545, -0.13761529326438904, -1.376920461654663, -0.8586074113845825, -1.7554904222488403, 1.9519882202148438, 0.47460460662841797, 0.6634631156921387, -1.092584490776062, -0.6703982353210449, 0.1047661304473877, 0.6505377292633057, -2.699831485748291, -0.21808218955993652, 0.3216586709022522, -0.17078381776809692, 1.2122448682785034, 1.9392249584197998, 1.172595500946045, -0.26300740242004395, -1.8924965858459473, 1.5894775390625, 0.2857455015182495, -2.7746455669403076, 0.1602863073348999, 1.0599961280822754, -1.0796219110488892, -3.3427865505218506, 0.7211382389068604, 0.3588929772377014, -0.9788607358932495, -1.7954320907592773, 0.004146873950958252, 3.35927677154541, 0.16986501216888428, 0.2629659175872803, -0.6948897838592529, 1.0756524801254272, -0.6728208065032959, -0.9075843691825867, 1.531498908996582, 1.5698686838150024, -0.28072696924209595, 3.2926571369171143, 1.9280147552490234, 0.2820216417312622, -0.02634561061859131, -0.0921546220779419, -2.119903326034546, -0.4379045069217682, -2.033146619796753, -0.6958318948745728, 1.2556837797164917, 0.05647122114896774, 1.0194342136383057, 0.9724306464195251, 0.9660004377365112, -1.0655244588851929, -1.3886923789978027, 1.263960599899292, -1.1278064250946045, -0.0015773773193359375, 0.389898419380188, -2.5404014587402344, 0.8487254977226257, -1.2894606590270996, 0.24657928943634033, -1.966090202331543, 0.06245351955294609, 2.1652016639709473, -0.10211867094039917, -0.2970919907093048, -1.7130595445632935, -0.9062132835388184, -0.3587283492088318, -0.15519008040428162, -0.14644493162631989, -1.556642770767212, 0.016011536121368408, 2.9811275005340576, -0.1744842529296875, -1.0697318315505981, 1.6509695053100586, -0.5287339687347412, 0.7590540647506714, -0.11123538017272949, -0.5383208394050598, 0.5804488658905029, 2.7051258087158203, 0.9511745572090149, 1.4152586460113525, -0.031811848282814026, -1.8043415546417236, -1.6711833477020264, 1.4897115230560303, -0.8461070656776428, 0.7493087649345398, 0.5713092684745789, 2.700852394104004, -0.5011893510818481, -0.34380197525024414, 1.394736647605896, 2.9451417922973633, 1.0772784948349, 0.05212101340293884, 1.2910959720611572, -0.4215833246707916, -1.7910276651382446, 0.13810774683952332, -2.2490689754486084, 0.8681813478469849, 1.4935592412948608, -1.50989830493927, -1.7973579168319702, 0.5952082872390747, 0.11066663265228271, 0.6816979646682739, 1.0932590961456299, -1.1831098794937134, -0.8431754112243652, 0.4105645716190338, 0.17006587982177734, -0.4266365170478821, -2.9991374015808105, -1.8308466672897339, 1.0178824663162231, 1.676162600517273, 1.8078670501708984, -1.372930884361267, 2.409604072570801, 0.9636883735656738, -2.731553554534912, -0.4555717408657074, 1.4384703636169434, 1.0571491718292236, 0.8011029958724976, -0.011992573738098145, 2.3406877517700195, 3.3755135536193848, 0.7129075527191162, 1.3854587078094482, 0.13633930683135986, -0.8272180557250977, 0.9795506000518799, 0.02331840619444847, 0.06460505723953247, -0.42844337224960327, 0.7041376829147339, 1.9224268198013306, 1.0486794710159302, -1.418941855430603, -0.992434561252594, 2.711167097091675, 1.669681429862976, 0.24002957344055176, -0.13461951911449432, -2.136244535446167, 0.1720820665359497, 2.0342299938201904, 0.03484344482421875, -0.5809021592140198, 2.3602347373962402, 0.17999684810638428, 0.6445326805114746, 1.3892887830734253, -1.0714049339294434, 0.9694778919219971, -0.41030240058898926, 1.6516492366790771, -0.33404579758644104, 0.1798335313796997, 1.6248143911361694, -1.357108473777771, 1.5486583709716797, -0.11869686841964722, 1.140920639038086, -1.2155632972717285, -0.8425748348236084, 0.016800925135612488, -0.976967453956604, -1.2357535362243652, -1.536785364151001, 3.6566073894500732, -0.8146002292633057, 2.3696999549865723, 1.1644511222839355, 2.1550140380859375, 2.9019434452056885, 1.2541420459747314, -2.580596923828125, -0.7895334362983704, -0.42680883407592773, -0.5395820140838623, -2.530010223388672, 0.5909909605979919, 1.0471138954162598, -2.6973841190338135, 1.7678836584091187, 1.2818442583084106, -0.6885709762573242, -3.079167604446411, 2.8750052452087402, 1.5168479681015015, 0.8023363947868347, 0.9550049304962158, -0.015400111675262451, 0.55828458070755, 1.8348208665847778, 0.162392258644104, 0.5927388668060303, -0.41949662566185, 1.409505844116211, 0.64142906665802, 1.6739857196807861, -1.8037642240524292, -0.134404718875885, 3.976206064224243, -1.6171613931655884, -0.554843544960022, -3.083782911300659, 2.6946513652801514, -0.36076420545578003, -1.1330187320709229, -2.695380210876465, -0.3481295108795166, -2.1833717823028564, -1.725373387336731, 2.264003038406372, -0.6182956099510193, 1.471838116645813, 0.7826094627380371, 1.9734022617340088, 0.8787279725074768, -0.6340230703353882, -0.9982212781906128, 3.4314932823181152, 1.431652307510376, 0.5063237547874451, 2.329796314239502, 2.245546817779541, 0.7548859119415283, -0.6413941383361816, -0.6105426549911499, -0.8280832767486572, -1.6515220403671265, -0.16847383975982666, -0.4011324942111969, -0.9333125948905945, 0.2967602610588074, -2.3794641494750977, 0.3402937650680542, -0.5950555801391602, -1.0982433557510376, -1.2274279594421387, 1.1633412837982178, 0.5807256102561951, -0.8333841562271118, 0.29287344217300415, -0.5934675931930542, 2.758068799972534, 0.20442356169223785, -1.894410490989685, 1.1123162508010864, 3.8469414710998535, 1.3486605882644653, -2.3285915851593018, -4.176692485809326, -0.6982154250144958, -0.5608245730400085, 2.475672960281372, 0.4685637652873993, -1.4696528911590576, -2.242116928100586, -1.5723285675048828, -2.763131618499756, 1.8781682252883911, -0.5040023922920227, -0.5185823440551758, 0.01001894474029541, 0.2339649200439453, 0.8119977712631226, 0.9714180827140808, 0.4938294291496277, -2.703582525253296, -0.9229686260223389, 0.694502592086792, -0.26955923438072205, 1.368188738822937, 1.6134324073791504, 2.246269464492798, 1.9567341804504395, 1.858412504196167, 0.7224006652832031, -1.6663587093353271, -0.11375650763511658, 1.942193865776062, -1.7906560897827148, 2.1699025630950928, -0.7984267473220825, -0.047730088233947754, -3.4066429138183594, 0.012966454029083252, -2.4020638465881348, 0.9952279329299927, -0.13732731342315674, 0.9722881317138672, 1.6132187843322754, -1.6179277896881104, -0.9788798093795776, -0.49302801489830017, -1.7072675228118896, 0.8727076053619385, -1.080322265625, -0.6541502475738525, -0.3256862759590149, -2.1617493629455566, -0.7597581148147583, -2.3729729652404785, 0.7981246709823608, -0.22951769828796387, -0.7864643335342407, -1.8734862804412842, 3.4058690071105957, 0.6516354084014893, 0.3655657172203064, 1.5015299320220947, -1.484894037246704, -0.2118770033121109, -1.1793632507324219, 1.1202276945114136, -0.5103745460510254, -0.5093141198158264, 1.5007959604263306, 0.5145567059516907, -2.155237913131714, 0.7892628908157349, -1.1637253761291504, 1.7570078372955322, -1.2155146598815918, -1.3199213743209839, 1.1089335680007935, -0.07218718528747559, 0.7294907569885254, 0.20034772157669067, 0.21116903424263, -0.3167802691459656, 0.3741039037704468, 0.6520566344261169, 0.6522765755653381, 2.4049484729766846, 0.8049910664558411, 1.5817660093307495, -1.5015788078308105, -0.2886645197868347, 0.7437680959701538, 0.47531455755233765, 1.6175649166107178, -0.3715314269065857, -2.070399761199951, 1.9089430570602417, -1.3905761241912842, -0.4520920217037201, 0.19093582034111023, 1.1501915454864502, -0.9829964637756348, 2.1422970294952393, 1.969855546951294, 2.5091724395751953, -0.4189075231552124, -1.4113414287567139, -1.2438302040100098, -1.0059118270874023, 2.0870046615600586, -0.6387271881103516, 0.5237905383110046, 0.3795267641544342, 1.0444698333740234, 1.8514690399169922, -0.6362066268920898, -1.1522774696350098, 1.5122864246368408, 1.2649104595184326, 1.567965030670166, 1.9918854236602783, 0.3280603289604187, -1.7820520401000977, 0.8786028623580933, 0.7044157981872559, 1.4755324125289917, 1.8890202045440674, -0.8300067782402039, -0.1519313007593155, -3.2931008338928223, 0.6182699203491211, 1.960557222366333, -0.6586955785751343, 0.5191473364830017, 1.5091251134872437, -1.3113747835159302, -1.4457319974899292, 1.2861289978027344, -0.4047743082046509, -3.2416062355041504, -0.3432697057723999, -0.6752838492393494, -0.6694737672805786, -0.9754146337509155, -0.8314477801322937, 2.311145067214966, -0.16616258025169373, -0.6924561858177185, -2.122187852859497, -0.25730055570602417, 0.04724442958831787, -1.7299988269805908, 0.1543608009815216, 1.211030125617981, -0.14674468338489532, 2.2375662326812744, 0.06644697487354279, -0.20730076730251312, -0.707696259021759, -2.1773340702056885, 0.007096350193023682, 1.8686916828155518, 1.2330262660980225, 0.2956523299217224, -1.6016432046890259, 2.624718189239502, 0.96547532081604, 1.0220192670822144, 0.47438371181488037, 0.4858933091163635, -0.3811449110507965, -1.466905951499939, 3.344296932220459, -1.2194756269454956, 1.441637635231018, -0.7414160966873169, -0.28057408332824707, -0.25651079416275024, 2.353593587875366, 3.5738582611083984, -0.6907593607902527, 0.6263895034790039, 0.5558104515075684, -1.1690151691436768, 0.943286657333374, 1.2176532745361328, 2.7164299488067627, 2.0692553520202637, 0.3034480810165405, 0.7493032813072205, -2.1346821784973145, 0.3239020109176636, 1.3913013935089111, -1.1185181140899658, 0.5162540078163147, 1.665327548980713, 0.5580639839172363, -0.37376338243484497, 0.1491336077451706, 2.5676567554473877, -2.3478457927703857, -0.8408417105674744, -2.053706169128418, -1.6012855768203735, 0.09807702898979187, -1.5885695219039917, -0.9288753271102905, -0.8626433610916138, -1.602782130241394, -1.3905425071716309, -0.8676903247833252, 1.4901769161224365, 2.565216064453125, 0.5410662889480591, -0.9583660960197449, -0.13695597648620605, 0.12426280975341797, -0.17012158036231995, 1.8607938289642334, -1.036214828491211, -1.4107095003128052, 1.6188195943832397, 1.8935643434524536, 0.3086121082305908, 0.42514705657958984, 1.7628387212753296, -0.2041682004928589, 0.8971535563468933, 0.7110518217086792, 2.2541425228118896, 2.9221768379211426, 0.8166013956069946, 2.253112316131592, -1.7455222606658936, 0.1114865243434906, -0.0764252245426178, 1.839367151260376, 0.08552521467208862, -2.2468223571777344, 1.1906116008758545, 0.03407865762710571, 0.046182796359062195, 1.2392644882202148, -0.9938555955886841, -1.1110976934432983, -1.04172682762146, 2.057799816131592, -0.7210515737533569, 0.10594397783279419, -0.25567197799682617, 1.657855749130249, 0.8220140933990479, 0.5905793905258179, -0.9307881593704224, -1.8737789392471313, 0.5638672113418579, -1.2817896604537964, 0.3885340690612793, 1.175261378288269, 1.4144742488861084, 2.620851516723633, 0.061652541160583496, 3.088406562805176, 2.3433988094329834, 1.4097707271575928, -0.4251123070716858, 1.477799892425537, 1.0822598934173584, -1.0618584156036377, -0.9316040873527527, 0.25357723236083984, 0.9335046410560608, -0.8227599859237671, 1.1666712760925293, -0.67777019739151, 2.655557155609131, 0.2446345090866089, -1.5247116088867188, 1.0490529537200928, -1.1050457954406738, 0.7172073721885681, 0.7369681000709534, -0.47826045751571655, -1.662743091583252, -0.8291821479797363, 3.1162333488464355, 1.9676120281219482, 1.6217708587646484, -3.4519126415252686, -0.49668216705322266, 0.3280234932899475, -0.57365483045578, -1.7810173034667969, -1.2847771644592285, 1.20309317111969, 1.5453294515609741, 1.7693629264831543, 2.5379133224487305, -0.8997966051101685, 1.0101200342178345, -0.5695511698722839, -0.11567699909210205, -1.2174698114395142, -0.23294872045516968, 0.44782406091690063, 2.0016698837280273, -0.7566468715667725, -0.5566847920417786, 2.214175224304199, 1.6287026405334473, 0.03290113806724548, -0.6939830183982849, -2.374807834625244, 0.05597865581512451, 0.5328430533409119, -0.12824425101280212, 0.14042234420776367, -1.6817376613616943, -0.21185848116874695, 0.012961626052856445, 0.7935094833374023, 1.9549344778060913, 0.15642477571964264, 0.5101479291915894, 0.1885879635810852, 0.055398158729076385, -0.25210464000701904, -4.880616188049316, -0.10670024156570435, 0.6263318061828613, -0.16639110445976257, -0.613073468208313, 1.0848963260650635, -0.39835304021835327, -0.7984204292297363, -0.8472291827201843, 2.183262348175049, -1.777737021446228, 1.4096333980560303, -0.4818558096885681, 0.30689650774002075, 0.7832464575767517, -0.7087924480438232, -0.7923300862312317, 2.3375043869018555, 1.1230493783950806, 1.0639005899429321, 1.1150896549224854, 1.8074219226837158, -0.03527712821960449, 0.8439956307411194, -1.157179594039917, -0.44172757863998413, 2.1055798530578613, -1.9354289770126343, -0.7302870750427246, -0.858261227607727, -0.13436484336853027, 0.45440009236335754, -0.6708794832229614, -1.7852177619934082, -0.790397047996521, -2.6822028160095215, -0.9552639722824097, 0.05738675594329834, 1.4128543138504028, 0.3573116362094879, 1.177093267440796, -1.0088794231414795, 0.7568129897117615, -0.49133962392807007, -0.3971545100212097, -1.5028190612792969, -1.232317566871643, 0.21734952926635742, 2.0446696281433105, -0.34402406215667725, 2.270817995071411, -0.5071924924850464, 2.6002306938171387, -2.072889804840088, -1.2970596551895142, 2.309889316558838, -1.2915012836456299, -0.41338834166526794, 1.2583519220352173, -0.8883141279220581, -2.0593531131744385, 0.5686923861503601, -1.3456951379776, -1.4733291864395142, 0.7081301212310791, -2.091862678527832, 0.49713951349258423, 2.452904462814331, 0.686937689781189, -2.752298355102539, -3.8148727416992188, -0.12154881656169891, 1.0759963989257812, 0.05756640434265137, 3.773160934448242, 1.1040576696395874, -0.08340075612068176, 0.8770769834518433, 3.3678574562072754, -3.1651875972747803, -0.7304936647415161, 0.07179972529411316, 1.5137344598770142, -1.2735462188720703, 1.9818224906921387, 1.3248332738876343, 2.2399048805236816, -0.49432623386383057, 0.12342974543571472, 0.2735118269920349, -0.3910908102989197, -2.0238730907440186, 1.9176208972930908, 0.6066575050354004, 1.4433367252349854, -2.076059103012085, 1.6808836460113525, 1.411575436592102, -1.6511281728744507, -1.4781713485717773, -2.5382766723632812, -1.5126452445983887, -0.2854638397693634, 1.4497984647750854, 0.8043504953384399, 0.37367820739746094, 0.29947254061698914, 1.1930370330810547, 1.571524739265442, 0.4099999666213989, 0.7335243821144104, -2.525284767150879, -0.8182893991470337, 1.1982463598251343, 0.365821897983551, -3.385237216949463, -1.2872967720031738, 2.539602756500244, 0.4128932058811188, -0.27679237723350525, -0.6557756662368774, -0.4497442841529846, 0.8167354464530945, -0.7081514596939087, -1.0194976329803467, -1.8667539358139038, -0.01691114902496338, -0.32027751207351685, 1.785158395767212, -1.3040848970413208, 0.2659469246864319, 1.2418981790542603, 0.004212021827697754, -1.2356833219528198, 1.3128167390823364, -1.6655616760253906, 1.2062277793884277, 0.8721878528594971, -0.33122390508651733, -0.9818845987319946, 0.9369702935218811, -0.15755565464496613, 2.217060089111328, 1.8263354301452637, -3.4052507877349854, 2.173062562942505, -0.9764959216117859, 0.16533571481704712, 0.09487307071685791, -0.8102151155471802, -0.5256233811378479, 2.6148343086242676, -2.4398562908172607, -0.8628467321395874, -1.2826310396194458, -1.3689632415771484, -0.30910202860832214, 0.5878139734268188, 0.5880220532417297, -2.4631505012512207, 1.1874744892120361, 0.24700331687927246, 0.021220073103904724, -0.3634601831436157, 2.1275534629821777, -1.2690982818603516, -1.3392045497894287, -0.14091166853904724, 0.42163458466529846, -2.3268537521362305, 1.162885308265686, -0.057913780212402344, 1.5335568189620972, 2.260063648223877, 0.0838172435760498, 1.4778691530227661, -2.0242764949798584, -3.590625286102295, -0.7943167686462402, 2.4520788192749023, 0.5677584409713745, 3.0419528484344482, 0.4046488404273987, 0.403250128030777, 0.11780726909637451, -1.375817060470581, -1.882624626159668, -0.32722848653793335, -1.0782625675201416, 0.18845891952514648, 0.9568088054656982, -2.2283644676208496, 0.7117857933044434, 2.0198752880096436, -1.724239468574524, 1.7634304761886597, -1.696529746055603, -1.0365184545516968, -0.26840144395828247, 0.9785034656524658, 4.007767200469971, 3.139655590057373, 1.6729724407196045, 2.153691530227661, -1.8838262557983398, -0.47815603017807007, -2.4275002479553223, -1.3417065143585205, 1.308944821357727, -1.4737368822097778, 1.0570944547653198, -2.088207960128784, 1.106736183166504, 0.8413037061691284, 0.9167076349258423, -0.4666576087474823, 0.7964782118797302, -0.448567658662796, 0.8480056524276733, 0.9398065209388733, -0.5083337426185608, 2.8147125244140625, -0.16535373032093048, -1.0140597820281982, -0.47845765948295593, -0.053807854652404785, 0.9450192451477051, 1.7926106452941895, 2.405275821685791, 1.963186502456665, -0.09517323970794678, -0.9147126078605652, 1.6641137599945068, -0.1132044792175293, -2.50577974319458, -2.5743231773376465, 0.7116363644599915, 1.9979828596115112, -0.8532953262329102, 0.8443182706832886, -0.1525549590587616, -0.7303425073623657, -1.3505377769470215, -0.24490416049957275, -0.6225413084030151, -2.4239320755004883, 2.0670173168182373, -1.4394872188568115, 2.1836798191070557, -0.425088107585907, 1.751409888267517, -1.6337171792984009, -0.2699325680732727, 2.018066167831421, 2.503931999206543, -0.6310454607009888, -2.642587661743164, 0.35619574785232544, -0.8036436438560486, 1.2660497426986694, -0.9190264940261841, 1.4652403593063354, 0.2590969502925873, 1.3427011966705322, 0.331105500459671, -1.4658212661743164, 1.1776816844940186, 1.3704273700714111, -0.06318965554237366, -0.3173792362213135, -1.0104162693023682, -0.9041738510131836, -0.6832116842269897, 0.37848037481307983, -1.0047776699066162, 0.5707234740257263, 0.4765624701976776, -1.5018353462219238, 0.1617826223373413, 0.8733572363853455, 0.5350049734115601, -1.2772691249847412, -1.8299083709716797, 0.19148105382919312, -0.6268874406814575, 0.17823642492294312, -0.35275402665138245, -1.704716682434082, -0.3931422829627991, 0.029071509838104248, 2.021141290664673, 0.259252667427063, 1.2292768955230713, -0.3051925301551819, -1.2489427328109741, 0.09447893500328064, -2.8387532234191895, 2.024249792098999, -0.29579219222068787, -0.39626240730285645, 2.4024012088775635, -0.8403037190437317, 1.9436637163162231, -0.8932662606239319, -0.5084362626075745, 0.6736831665039062, -0.7034695148468018, 2.0948565006256104, -0.39899134635925293, -1.5392110347747803, 1.3923466205596924, -0.09284466505050659, -1.1047558784484863, 1.1759796142578125, 1.5023478269577026, -0.26507312059402466, -0.6025921702384949, -1.3426088094711304, -0.35104086995124817, -0.5982128977775574, 0.4301677644252777, 1.3170061111450195, -1.2072733640670776, 3.6047847270965576, 0.8736798763275146, 0.5131988525390625, 0.08428674936294556, 1.033402681350708, -2.855517625808716, 0.6334927082061768], [-1.1004197597503662, 1.4852631092071533, -0.7077193260192871, 1.9938730001449585, 1.0782161951065063, 0.28522276878356934, 1.1147079467773438, -2.9858250617980957, -0.4343493580818176, -0.23769859969615936, -0.26008570194244385, -1.2564332485198975, 1.2692980766296387, 1.7111377716064453, -1.8410453796386719, -0.6341938972473145, -1.4871480464935303, 1.736402153968811, 0.01787668466567993, 0.5769720673561096, -0.844645619392395, 1.1791090965270996, 1.7181581258773804, 0.05900275707244873, 0.8074162602424622, 1.1377240419387817, 0.6422626376152039, -0.8147447109222412, 0.36082005500793457, -0.5429655909538269, 0.9382885694503784, -1.05007004737854, -3.0215506553649902, -1.190955638885498, -0.19833093881607056, -0.5829662084579468, 1.7911877632141113, -1.050744652748108, 1.6128028631210327, -3.1759860515594482, -1.5545392036437988, -0.05944371223449707, 1.1890274286270142, -0.5208711624145508, -1.4859801530838013, 0.8419821262359619, -1.186572790145874, -0.32024282217025757, -0.20140355825424194, -0.19037669897079468, 1.995337724685669, 0.8904513120651245, -1.1634308099746704, 0.5314847230911255, -1.452361822128296, -1.4821205139160156, -0.27011239528656006, -1.7325494289398193, 0.25224149227142334, 1.138985276222229, -1.2096421718597412, -0.6491718292236328, 0.038154542446136475, -0.48724478483200073, -1.585776686668396, 1.608065128326416, -3.8185648918151855, 2.9243509769439697, 1.240918517112732, -1.8227671384811401, 1.6728870868682861, 1.4288280010223389, -0.7273088097572327, -0.8057235479354858, 1.723912239074707, -1.4152355194091797, 2.0968940258026123, 0.019888371229171753, -1.0905472040176392, -1.1768556833267212, 0.23874682188034058, 1.8514974117279053, -1.024237871170044, -0.009260892868041992, 0.1515737920999527, -0.3314242959022522, -2.0189311504364014, -0.14662213623523712, -0.7317956686019897, -0.8248246312141418, 1.0309929847717285, -1.372514247894287, -0.6044082045555115, -0.4382585883140564, 0.4072909355163574, -1.4316062927246094, 1.5217384099960327, 1.7484807968139648, 0.4622596800327301, -0.5392341017723083, 2.116314649581909, 0.9837204813957214, 0.15819266438484192, -0.0681852251291275, 0.05211186408996582, -0.4541161060333252, -2.394401788711548, -1.9273350238800049, -0.03242814540863037, 2.225341558456421, -1.7449839115142822, -0.44867435097694397, 1.7073886394500732, -2.4450902938842773, 0.5212039351463318, 0.049557268619537354, -1.8561158180236816, -0.9216245412826538, -0.29414090514183044, -0.24301016330718994, 1.447629451751709, 0.23261040449142456, -1.0528813600540161, 0.8126391172409058, 2.36055326461792, -0.9793787002563477, -1.2940688133239746, -1.4555470943450928, 2.2098159790039062, -0.19342249631881714, -1.9134671688079834, 0.15064260363578796, -0.9927796125411987, 0.8356704711914062, 0.9400082230567932, 0.6472568511962891, 0.41084086894989014, 0.31998562812805176, -0.9777380228042603, -0.23956316709518433, -0.7560800909996033, 0.793442964553833, -1.66639244556427, -0.050646066665649414, 4.686033248901367, -0.2398795485496521, -1.0292375087738037, -0.648159384727478, -1.6028586626052856, 1.3783411979675293, 2.048098087310791, 0.4219319820404053, -3.2860684394836426, -0.6292656660079956, -0.84492027759552, -1.2801401615142822, -1.7626177072525024, 0.20008914172649384, 0.6565786600112915, -1.3033673763275146, -2.6213631629943848, -0.22691470384597778, -1.1522620916366577, -2.0649008750915527, 0.6548861265182495, -1.887501835823059, 1.5122277736663818, 1.041854739189148, 0.878113865852356, 1.7303297519683838, -1.0960867404937744, -1.5856019258499146, 2.1211018562316895, -1.2575668096542358, -1.0583816766738892, 0.016189947724342346, 2.4748308658599854, -1.1290373802185059, -0.21174713969230652, -2.3362722396850586, -0.7061490416526794, 0.005762219429016113, 0.5174119472503662, 0.7249032258987427, -0.6756858825683594, -1.1678967475891113, 2.2748453617095947, -0.335435688495636, 0.48630034923553467, -0.18718111515045166, -1.4231274127960205, 0.06199240684509277, 0.0007399320602416992, -1.736404299736023, -0.5137617588043213, 0.42021751403808594, 2.0171878337860107, 1.3210108280181885, 1.1044762134552002, -0.05205986276268959, -0.2063373625278473, 0.37642067670822144, -0.5876162052154541, -1.2443585395812988, -0.5487326979637146, 0.5936177968978882, -2.6599602699279785, -1.0363832712173462, -1.6838839054107666, -0.6021095514297485, -2.4161601066589355, -1.4001975059509277, 1.1061452627182007, -0.7211713194847107, -0.9730918407440186, -1.0663185119628906, -4.0235514640808105, 0.07551723718643188, -0.786231517791748, -0.4638991951942444, 1.057679533958435, -0.42598071694374084, -0.01403200626373291, -0.6249540448188782, -1.2010703086853027, -1.245374321937561, 1.8545066118240356, -1.7343854904174805, 1.30547297000885, -0.24801620841026306, 0.6342126131057739, -0.614958643913269, -1.9438235759735107, -0.9614258408546448, -0.5988584160804749, 2.0192978382110596, -1.0155514478683472, -1.2019944190979004, 2.2645790576934814, -0.4169975221157074, -1.6056996583938599, -1.6894798278808594, 1.9090309143066406, -0.5068812370300293, 0.2878867983818054, -1.5365686416625977, -0.9097260236740112, 0.3940640985965729, -1.9393410682678223, 0.6760088801383972, -0.8513545989990234, 2.083712100982666, 1.1601358652114868, -0.6317354440689087, -2.9524357318878174, 1.1849267482757568, -0.6519525051116943, 0.8936214447021484, 0.11433547735214233, -0.47025662660598755, 0.5697296857833862, -0.06862366199493408, -2.1106739044189453, -0.2683160901069641, -1.0190638303756714, -1.2033718824386597, 0.40643730759620667, 1.229947566986084, -1.2992781400680542, -1.7552590370178223, 1.8343216180801392, 0.49518442153930664, 1.013162612915039, 2.269841194152832, 0.0023189783096313477, 0.5266554951667786, -2.2620744705200195, -0.7604718208312988, -0.7300834059715271, 0.6159805059432983, -0.6347261071205139, 0.15873217582702637, -1.0332025289535522, -0.5923282504081726, -0.19008660316467285, 0.446938157081604, 0.572695255279541, -0.48246288299560547, -2.459990978240967, 1.4744242429733276, 2.7566568851470947, 0.4861863851547241, -0.9306249618530273, 0.5644962191581726, -3.391052007675171, 1.8709348440170288, 1.0704171657562256, 3.989147901535034, 0.9662426114082336, -0.896144688129425, -1.116333246231079, 1.2753541469573975, 0.48704275488853455, -1.0951128005981445, -0.28146588802337646, 0.4510253369808197, -1.699586272239685, -1.573137640953064, 0.48724210262298584, -1.7237493991851807, 1.2374954223632812, 0.998076856136322, 0.07829654961824417, -0.46550875902175903, -0.9398294687271118, 2.166682720184326, 0.18188557028770447, -0.40991976857185364, -2.196575880050659, -1.73819899559021, 1.0062479972839355, -1.2245557308197021, -1.5174479484558105, 0.6423142552375793, 1.7963647842407227, 1.7546993494033813, 2.596062660217285, 0.774122416973114, 0.13492833077907562, -0.16067127883434296, 2.0263137817382812, 0.4346180558204651, 0.02994835376739502, 0.8455794453620911, -1.0922763347625732, 0.48363667726516724, -0.28836995363235474, 0.09723685681819916, -0.930735170841217, 0.6728505492210388, 1.0745561122894287, -2.3884196281433105, -2.7057764530181885, 2.5586159229278564, -0.7409833073616028, -0.9499436020851135, -1.9869422912597656, -0.9910978078842163, 0.6791342496871948, -0.8323556184768677, 0.7880424857139587, -0.3450607657432556, -0.18303073942661285, -2.000072717666626, 0.8612459897994995, 0.2974521517753601, -2.01651668548584, 2.3588099479675293, -0.5760524868965149, -0.42303329706192017, 2.2272305488586426, 0.30574166774749756, 0.34340521693229675, 2.189112424850464, 0.3825453817844391, 0.3240307867527008, -0.2661436200141907, -0.5339325666427612, 1.010481834411621, 1.3433328866958618, -0.7025877237319946, -1.9136046171188354, 1.0832531452178955, -1.4655067920684814, -1.864046573638916, -0.2706265151500702, -0.2637312412261963, -0.6935840845108032, 0.11491286754608154, -1.3514654636383057, 0.6444643139839172, -1.6422486305236816, 1.5940688848495483, 0.018599897623062134, -0.12846291065216064, -0.10586988925933838, 2.0448176860809326, 0.0009723156690597534, -1.2484865188598633, -0.26909804344177246, -0.763625979423523, -0.6055119633674622, 1.6622984409332275, 0.7593652009963989, 1.16777503490448, 2.032453775405884, -2.0750226974487305, -2.5804247856140137, 2.053395986557007, -0.7328411340713501, -0.19986051321029663, 0.6297966241836548, 0.8011660575866699, 0.7402399182319641, 0.030504226684570312, -1.0744507312774658, 0.42693525552749634, 0.47093304991722107, -1.3588125705718994, 2.3199145793914795, 0.31187307834625244, 0.4391144812107086, 1.552809238433838, -1.497117042541504, 1.2118957042694092, -2.26505184173584, 0.3163447678089142, -0.9619856476783752, 1.282638430595398, -0.17914101481437683, 1.1986173391342163, -0.6032510995864868, 0.3131089508533478, -0.0891830325126648, 1.5194263458251953, -0.004842519760131836, -1.9187737703323364, 1.2308086156845093, 0.7639887928962708, 0.5745967030525208, -0.6460088491439819, 0.7904129028320312, -0.49291378259658813, 0.4252522587776184, 1.5462331771850586, 0.06338340044021606, -0.9069581627845764, 0.3139345645904541, 0.5559084415435791, 0.5201225280761719, 0.7241798043251038, -0.3507016897201538, 2.422921657562256, -0.09788713604211807, -1.8618589639663696, 0.8705071210861206, -1.0822776556015015, 2.053596019744873, -0.983485996723175, 0.24113380908966064, -1.734882116317749, 0.6419400572776794, 1.1620439291000366, -0.016068577766418457, 0.7440671920776367, -0.7460936307907104, 0.6068315505981445, -1.5918567180633545, -2.2813448905944824, -1.7069871425628662, -0.979884684085846, 0.22198456525802612, -1.8792071342468262, -2.430021047592163, 1.5100055932998657, 0.6246956586837769, -1.4096850156784058, -0.17184999585151672, -0.26775795221328735, 0.36422502994537354, 0.9842521548271179, -0.47221511602401733, -1.7186107635498047, 0.11074179410934448, 0.05430464446544647, -0.3442152142524719, -0.4409142732620239, -1.1152565479278564, 2.6154184341430664, -0.7737187743186951, 1.359324336051941, 1.0602717399597168, 2.8880867958068848, -1.3254761695861816, 1.6953954696655273, 0.09510861337184906, -0.3786197900772095, -0.5655832290649414, -1.0800586938858032, -1.8922022581100464, 0.24956762790679932, 0.46556201577186584, 1.3508325815200806, 1.7673356533050537, 1.0305664539337158, 0.15448245406150818, 2.514416217803955, -0.1456890106201172, -1.6421482563018799, 0.34006449580192566, 1.4077941179275513, -0.4530392289161682, 0.5020353198051453, 2.127460241317749, 2.004635810852051, -1.3281804323196411, 1.2232471704483032, -2.3839714527130127, -0.43561121821403503, 1.8400235176086426, -0.4819739758968353, 0.8680515885353088, 1.7476155757904053, -0.035218045115470886, 1.9376742839813232, -0.12083455920219421, -0.4736199378967285, -1.010642409324646, -1.3911746740341187, 0.5287450551986694, -0.19729876518249512, 1.7428687810897827, 1.8898359537124634, 1.044862985610962, -0.21455377340316772, 2.7079946994781494, 0.5947443246841431, -1.2234641313552856, -2.9334423542022705, 0.723373532295227, 0.8794383406639099, 2.231398582458496, -0.6165696978569031, 0.31735914945602417, -1.0511739253997803, 0.692784309387207, 1.9916152954101562, -1.1673697233200073, 2.2904319763183594, -0.8122555613517761, 1.2346971035003662, -0.5763394832611084, -1.127152681350708, -1.7571438550949097, 0.08167842030525208, -0.9479207992553711, 0.6391001343727112, -0.9123422503471375, -0.31508347392082214, -3.107086658477783, 2.846190929412842, -1.781334400177002, 1.919958233833313, 0.27093660831451416, -0.36870843172073364, 0.8765191435813904, -0.4033340811729431, -0.6010453701019287, -2.4361135959625244, 0.6873774528503418, 2.933724880218506, -1.1495451927185059, 1.3756341934204102, -2.7896728515625, 0.49055251479148865, -0.06648018956184387, 1.0207293033599854, 3.2786617279052734, 0.1461542397737503, 0.6726412773132324, 1.3032556772232056, 1.148171067237854, -0.010649621486663818, -0.39391636848449707, -3.3370704650878906, 0.4738203287124634, -0.6798754334449768, -0.6397388577461243, 0.3258717954158783, 1.8177540302276611, 2.6589272022247314, -2.181947708129883, 2.9952800273895264, -0.2772890329360962, 1.0348541736602783, 0.3667674660682678, -1.919555425643921, -0.38975706696510315, 1.391921043395996, -1.3119065761566162, 0.3291444480419159, -1.1894837617874146, -2.400371551513672, -0.7127544283866882, -2.0452239513397217, -3.06895112991333, -0.5634044408798218, 1.102552890777588, -1.4263352155685425, -0.11131429672241211, 1.2748374938964844, -2.585041046142578, -0.21819454431533813, -0.4681037664413452, 0.46381840109825134, -0.11580909043550491, 1.531972885131836, -1.3713436126708984, -1.1859644651412964, -1.2973251342773438, 0.29061225056648254, 2.73093843460083, -3.0648043155670166, 1.300908088684082, 1.0355674028396606, 0.40696483850479126, 0.2125697135925293, -1.0143481492996216, -3.224266290664673, -0.4674270749092102, -0.7314218878746033, -1.3807051181793213, 1.7932859659194946, 2.9974989891052246, -0.9518300294876099, 1.742580771446228, 0.5045766830444336, -1.4420850276947021, 0.49537378549575806, -0.0914069414138794, -0.3476305305957794, 0.9967608451843262, 0.2991677522659302, 1.4810861349105835, -0.832472562789917, -2.69864821434021, 1.9370815753936768, -0.8015816807746887, 2.7442078590393066, -1.115723967552185, -1.996990442276001, 0.0871923565864563, -2.9785022735595703, 2.969557762145996, 1.7593646049499512, -2.3120028972625732, -2.7638492584228516, 1.204170823097229, 1.2097505331039429, -0.8393210172653198, 0.44555801153182983, -0.4747929573059082, 1.8033629655838013, -0.9751176834106445, -2.710319995880127, -2.2284345626831055, 0.6561233401298523, -3.086151123046875, -0.9765064716339111, 2.479943037033081, 0.36042463779449463, 0.26740801334381104, -0.7129542827606201, -0.27079105377197266, -0.2660359740257263, 1.694291114807129, 1.5985887050628662, -1.2156996726989746, 1.4576165676116943, -1.2956945896148682, 0.29550471901893616, 1.558678150177002, -0.8350372314453125, 2.701875686645508, 0.3498897850513458, -0.17586058378219604, -0.31625598669052124, -1.2892417907714844, 1.9804115295410156, -2.5335474014282227, 0.4009973406791687, -0.2073715329170227, -2.5417685508728027, 1.137323021888733, 0.07754509150981903, -1.3573665618896484, -0.3893570899963379, 0.14079034328460693, 0.07983005046844482, -1.421433925628662, -0.5987103581428528, 0.0036150217056274414, 0.23667486011981964, -0.1648818850517273, 1.4480541944503784, 0.31537044048309326, 3.002243995666504, 1.7426111698150635, 2.1696882247924805, -1.104304552078247, -2.0130152702331543, -1.6457549333572388, 0.017954111099243164, -0.8117862343788147, -1.946275234222412, 3.292886972427368, 0.7643114924430847, 0.05612906813621521, -1.3229660987854004, -1.8470100164413452, 1.5530312061309814, -2.232358455657959, 0.06477862596511841, -2.3694372177124023, -2.281365156173706, -0.02780762128531933, -0.9420070648193359, 2.4849884510040283, 0.9347448945045471, -0.5549907684326172, 0.26072072982788086, -0.4986720681190491, -1.7567042112350464, -0.6318287253379822, -0.2602071166038513, 0.46990108489990234, 0.05710500478744507, 0.6483790874481201, 0.2274947166442871, -1.6633621454238892, 0.8174813389778137, -0.7260085940361023, 1.8418869972229004, 1.1489194631576538, -0.1722726821899414, -0.5369448065757751, -1.1592429876327515, 1.1457819938659668, 0.5044012665748596, -1.1982824802398682, 0.004034832119941711, -1.8958755731582642, 1.5638282299041748, 2.692949056625366, -1.0797810554504395, -0.43615078926086426, -0.9346765279769897, 0.48208484053611755, -0.5689541101455688, 1.4411827325820923, 1.4827845096588135, -1.0999455451965332, -1.183241844177246, -0.7093051671981812, 0.2861090302467346, 0.5043795704841614, -1.902472734451294, 0.6292558312416077, -2.1297197341918945, -0.42701107263565063, -0.22994491457939148, 0.35389482975006104, -2.4878878593444824, -4.020695686340332, -0.8484485149383545, 0.006657719612121582, 0.9898338317871094, -0.4754854440689087, 0.7991846203804016, 0.8262180089950562, 0.0064476728439331055, 1.4688987731933594, 0.5921196341514587, -0.3624511957168579, 0.6213943958282471, -1.6059969663619995, 0.30238789319992065, -0.860207200050354, -1.541558027267456, 2.738560676574707, -0.08958154916763306, 2.1523232460021973, 1.1268415451049805, 1.9492096900939941, -1.531779170036316, 1.132066249847412, -0.6174442768096924, -0.8457355499267578, 0.8686225414276123, -1.1017298698425293, 0.8774369955062866, -0.8233349323272705, 0.7944096326828003, -0.5947454571723938, 0.5284396409988403, -1.0094116926193237, 1.6619030237197876, 1.5241703987121582, 1.5561892986297607, 1.0113399028778076, 0.5143167972564697, -0.9255606532096863, -1.2110297679901123, 0.08117401599884033, -0.198192298412323, -3.469261884689331, 2.126481533050537, 0.6249440312385559, 1.327523946762085, -1.9169549942016602, 1.0919898748397827, 1.1482409238815308, 0.2580203413963318, 0.5531772971153259, 0.4873715341091156, -0.2219635248184204, -1.00674569606781, -0.6151270270347595, 0.587288498878479, 2.0972306728363037, 0.028131186962127686, -2.1900694370269775, -1.4174978733062744, -0.9331537485122681, -0.9449329972267151, -2.4211184978485107, 0.3996662497520447, 0.4058111906051636, 0.7770792841911316, -1.119390606880188, 0.6511813402175903, -0.48015570640563965, -1.0481128692626953, -4.299997329711914, -0.8656961917877197, 0.8314546346664429, -0.8833231925964355, 1.349395990371704, -1.028342843055725, -0.11384320259094238, 0.015056730248034, -0.18947356939315796, 0.6564399003982544, 1.2945988178253174, -1.0291216373443604, -1.0456620454788208, -0.6429785490036011, 0.24331355094909668, -0.32151973247528076, 1.1824356317520142, 0.533781886100769, 0.6613460183143616, -0.3708195090293884, -0.5605023503303528, 1.5381393432617188, 0.5059809684753418, -0.8210094571113586, -0.5589162111282349, -2.96329927444458, 0.0062544941902160645, 1.2224164009094238, -0.06187915802001953, 1.0477081537246704, 0.31252601742744446, -0.406319260597229, 1.5894031524658203, 1.0817588567733765, -0.5064471364021301, 2.449272632598877, -0.5530091524124146, 0.8702146410942078, 1.245842456817627, -2.289362907409668, -0.17497998476028442, -2.2529897689819336, -1.2015317678451538, 0.38007551431655884, -1.12809157371521, 0.8115367293357849, -0.8818967342376709, -2.016200304031372, -0.22973071038722992, -0.9624038934707642, 0.1973295956850052, 0.34684890508651733, 2.325251817703247, 0.7893950939178467, 1.3071718215942383, -0.9200501441955566, 1.5393717288970947, 1.4845869541168213, -0.23728474974632263, -2.4271814823150635, 0.9402339458465576, 0.5862008333206177, -1.5297057628631592, -0.48427507281303406, -2.0070748329162598, -0.7164705395698547, 0.8776669502258301, -2.2690539360046387, -0.16139793395996094, 0.744868278503418, 0.7914414405822754, 0.22203883528709412, -1.4978151321411133, 0.1079631820321083, -0.8477731347084045, 2.121856212615967, -1.584966778755188, -0.08981335163116455, 1.321123480796814, -1.4441659450531006, -1.7710520029067993, -2.8441600799560547, -0.4825281798839569, 1.866024374961853, 0.6385349035263062, -0.04608839750289917, -1.005902886390686, 1.9329164028167725, 0.5820234417915344, 2.702471971511841, 1.6694071292877197, 0.3860219120979309, 0.4897755980491638, 1.0945217609405518, -1.0139284133911133, 1.1970396041870117, -1.0671734809875488, 0.4104287028312683, 3.14809513092041, 0.741225004196167, 3.874722719192505, -2.6737775802612305, -1.1091883182525635, 0.993342399597168, 0.02519974112510681, 1.4002118110656738, -2.7845571041107178, 0.3085078001022339, -0.23893582820892334, 0.5464180707931519, 1.9607625007629395, 0.6993595957756042, 0.7538491487503052, 0.5231186151504517, 1.4366860389709473, 1.6737052202224731, -0.041520342230796814, -0.7771561145782471, -0.6862239837646484, -2.2949376106262207, -0.9225084781646729, -1.8112375736236572, 0.8659481406211853, -0.18655967712402344, 1.039566993713379, 0.2791348695755005, -0.40842992067337036, -0.9259005784988403, 2.4713821411132812, -2.7676024436950684, -0.6596245765686035, 1.9101946353912354, 0.6920850276947021, 1.437082052230835, -0.4595968425273895, -1.219563364982605, -0.48144373297691345, 0.3327620029449463, -2.304030418395996, 0.6631084680557251, -1.5628613233566284, 0.34025925397872925, 2.8202993869781494, 0.25850701332092285, 1.87860107421875, -1.1266582012176514, 1.0472224950790405, -0.15436440706253052, -1.1429359912872314, -0.5818156003952026, 0.8588790893554688, -1.5216896533966064, 0.4540773928165436, -1.9116631746292114, -0.17673397064208984, 3.5490589141845703, -0.9298848509788513, -1.2658896446228027, -0.44190865755081177, 0.6907638311386108, 2.2942562103271484, -1.472515344619751, -1.5681772232055664, -0.4697318971157074, 2.3082823753356934, 1.32657790184021, 0.004529118537902832, 0.9571413397789001, -0.41987693309783936, 0.5116394758224487, 0.7976142168045044, 1.9647619724273682, -0.32163965702056885, 1.8890280723571777, -2.4130449295043945, 0.52208012342453, -0.9000763893127441, 0.2928820848464966, 1.3877129554748535, -0.005483701825141907, 0.09247277677059174, -2.1622157096862793, 0.6922346353530884, 2.095745801925659, 1.3632500171661377, -0.7294636368751526, 0.26536357402801514, -2.5414836406707764, 2.2694766521453857, -0.6293807029724121, -1.1932411193847656, -0.01696765422821045, -0.2640307545661926, -1.0126538276672363, -2.845914602279663, 0.7001982927322388, 0.661885142326355, -1.845489501953125, 1.62446928024292, 0.3175293803215027, -1.1115809679031372, 0.4153897762298584, -0.15061986446380615, 1.2156026363372803, 0.08268025517463684, 0.43465399742126465, 0.40816766023635864, 1.6479939222335815, 2.0943942070007324, 1.4942291975021362, 0.8544873595237732, -1.7521177530288696, -2.3881783485412598, 3.1061654090881348, 1.8812952041625977, 0.05453544855117798, -0.10481786727905273, -0.8785004019737244, 1.192647099494934, 1.6262749433517456, 0.5417762398719788, 0.3777381479740143, -0.16085422039031982, -0.9790134429931641, -1.4318164587020874, -0.5983116030693054, -3.2620105743408203, -0.7504873275756836, -2.8466904163360596, -0.15782403945922852, 0.7278023958206177, -1.2319837808609009, 1.4492108821868896, 2.270967960357666, -1.209372639656067, -1.2446374893188477, 0.732104480266571, -1.575594425201416, -0.5067233443260193, -0.13305920362472534, 0.43443265557289124, -0.5629480481147766, -2.0832924842834473, -2.248953104019165, -0.1068832278251648, -0.1200217604637146, -2.0112955570220947, -0.967243492603302, 2.872403860092163, -4.431288719177246, -0.04884016513824463, -0.437600702047348, -1.212886095046997, 0.3393051028251648, 0.7940505146980286, 1.5694717168807983, -0.233107328414917, 0.6249157786369324, -1.3979003429412842, 0.6553236246109009, 2.7865254878997803, -0.3170585036277771, -0.8576750755310059, 1.548400640487671, -2.396406650543213, -0.46486854553222656, -1.5185630321502686, -0.05225633084774017, -1.5778300762176514, 0.6427396535873413, -1.056877851486206, 3.0534229278564453, -1.9038751125335693, 1.0595524311065674, 0.30059438943862915, -0.8440954685211182, 1.0086209774017334, 0.7628726959228516, -0.5971689224243164, 0.9294368028640747, 0.9607821702957153, 1.1832183599472046, 0.012197459116578102, 2.295917272567749, 0.16250669956207275, 1.396286129951477, 0.5597504377365112, 2.676145553588867, -2.0034351348876953, -2.2504091262817383, -1.429443120956421, -1.8561947345733643, -0.3935174345970154, 1.2618099451065063, 0.9055150747299194, -3.561725616455078, 0.6663657426834106, -0.31208258867263794, 0.8144124746322632, 2.195943832397461, -0.0959135890007019, 1.8607584238052368, 0.4920634627342224, 0.0005296468734741211, -0.11596617102622986, 0.1985642910003662, -1.1807103157043457, -1.7151226997375488, -0.6752748489379883, -0.3944952189922333, -3.2673585414886475, 0.49996599555015564, -0.8878160119056702, 0.6076123714447021, 1.552885890007019, 1.3584142923355103, 1.4824016094207764, 1.3066165447235107, 0.8838332891464233, 1.1128957271575928, -1.4880928993225098, -1.3430287837982178, -0.9072189331054688, 1.1883940696716309, 1.5661191940307617, 3.547693967819214, -2.0079164505004883, 1.0177392959594727, -3.0691163539886475, 1.6668840646743774, -0.7663638591766357, -0.33713483810424805, -1.4281734228134155, -1.6023693084716797, 0.07425868511199951, -0.3079082667827606, -0.02297966182231903, -0.687545120716095, -0.6673262119293213, 1.110556960105896, -0.852185845375061, -1.5657954216003418, -0.521915078163147, 0.8316324949264526, -1.2716670036315918, 1.7434853315353394, 0.7466191649436951, -0.5357035398483276, -3.021468162536621, 1.94600248336792, -0.17095160484313965, 1.074174165725708, -0.02256298065185547, 1.5234404802322388, -2.4344980716705322, -1.702276349067688, 0.29366010427474976, -1.9019814729690552, -1.115661382675171, 1.7788807153701782, 2.6134543418884277, -0.06890997290611267, -0.0025083720684051514, -1.713784098625183, -1.0161430835723877, 0.88871830701828, -1.24558424949646, 1.0701152086257935, 0.24959611892700195, 1.6915534734725952, -0.3587977886199951, -0.17293426394462585, 0.5816779732704163, -0.657764196395874, -1.679072380065918, -1.9440932273864746, -0.8426255583763123, -0.7867374420166016, 0.012789726257324219, 1.9744815826416016, 0.6789394021034241, -3.280932903289795, 0.1199171245098114, 0.12075325846672058, -2.3870511054992676, -0.5463485717773438, 0.04463696479797363, -0.36055076122283936, 2.582663059234619, 2.520559787750244, -0.2580486536026001, -0.03670415282249451, -0.4495939612388611, 2.0330352783203125, -2.1189308166503906, -0.3173943758010864, 0.22331103682518005, -0.3978537321090698, -1.6912823915481567, -1.6773555278778076, -0.815809965133667, 0.2869306206703186, -1.5449063777923584, 0.3379330039024353, -1.0110231637954712, 0.2574896812438965, -0.44317686557769775, -3.1636877059936523, 0.760398805141449, -1.6532524824142456, 0.5086838603019714, -1.2085742950439453, -0.29219740629196167, 0.12091034650802612, 0.9027380347251892, -0.02811509370803833, -0.3184320330619812, 0.01112377643585205, 0.48068004846572876, -1.4391731023788452, 1.1610521078109741, -0.011631369590759277, 0.770084023475647, -1.3065218925476074, -0.5622813105583191, 1.3648786544799805, -0.17535489797592163, 0.8058967590332031, -0.8822094798088074, 0.10116231441497803, -2.3454127311706543, 0.435062050819397, 1.142438530921936, -0.07142174243927002, -0.304279088973999, -0.654249906539917, 1.4556610584259033, 0.46657228469848633, 1.3396658897399902, -1.713759422302246, 1.188625693321228, -1.117126226425171, -0.5626821517944336, 1.3195300102233887, -1.8425273895263672, 1.5843713283538818, 1.0797672271728516, 1.1056278944015503, 4.450291633605957, -0.6100851893424988, -1.0806069374084473, 0.23917502164840698, 1.2743741273880005, 0.9858303070068359, -0.695368230342865, -0.6698870658874512, -0.6461800336837769, 0.6909654140472412, 1.9915484189987183, 1.740976333618164, 1.6661882400512695, -0.20060451328754425, 1.5534820556640625, 1.7447102069854736, 0.18950265645980835, -2.191481113433838, -2.373565196990967, -0.1918271780014038, -0.08331771194934845, 0.5336630344390869, -0.360820472240448, 1.2215496301651, 1.5510668754577637, -1.2144296169281006, -0.4508921504020691, -1.9180774688720703, 0.2498716115951538, -0.8478044867515564, -0.8429554104804993, -2.028329610824585, 0.8710469007492065, -0.08425891399383545, -1.4588022232055664, -0.06751120090484619, -3.095639228820801, -1.8228414058685303, 0.11569526046514511, -0.32659849524497986, -1.5836892127990723, -0.3461416959762573, 3.249397039413452, -0.8285396099090576, 1.808093786239624, 0.4896337389945984, 2.040489673614502, 2.263941526412964, -0.3502182960510254, 1.2260661125183105, 2.8508710861206055, 1.585881233215332, -0.6231740713119507, 0.03919115662574768, 0.12881900370121002, 1.2284833192825317, 1.2093710899353027, -0.7518537044525146, -1.0056766271591187, 0.20158615708351135, -2.7536301612854004, 0.8265613317489624, -0.825994610786438, -1.0406761169433594, -1.0332369804382324, 1.1885781288146973, 1.5159380435943604, -1.2941455841064453, 0.6576015949249268, -0.45092064142227173, 1.753702163696289, -0.19586940109729767, 2.2860641479492188, -0.681520938873291, -3.0394105911254883, 0.49346405267715454, -1.0341626405715942, -1.4886271953582764, -0.5412120819091797, 0.6457864046096802, -1.784869909286499, -0.7608155012130737, 0.6232779026031494, -0.13399678468704224, 0.41230636835098267, 0.8924441337585449, -1.2497361898422241, 0.5504662990570068, -1.2657294273376465, -1.6475801467895508, -2.2168939113616943, 0.21492069959640503, 1.2142372131347656, -1.1110310554504395, -0.7593792676925659, 0.7427362203598022, -0.16590282320976257, 0.07144749164581299, -1.0239595174789429, 0.1939665973186493, 2.186295509338379, 1.4291110038757324, 0.05161432921886444, 0.5700578689575195, -0.15248757600784302, -1.5903681516647339, 0.7572678327560425, 0.01830369234085083, -1.5695537328720093, 0.9093905687332153, -1.3014687299728394, 0.24769216775894165, -3.927032470703125, 0.5122939348220825, 2.7663862705230713, -0.8503073453903198, -2.0328338146209717, 2.2218477725982666, 0.4433537721633911, -0.8009538650512695, -1.1651242971420288, 0.1663346290588379, 1.051264762878418, 0.9742552638053894, -0.39778217673301697, 1.8419959545135498, 0.8774659633636475, 0.4609001874923706, 0.6836743354797363, 2.4894800186157227, 1.1995930671691895, 1.186228632926941, -0.012210607528686523, -1.8438539505004883, 0.7621929049491882, -0.6943609714508057, 0.5527520179748535, -1.5851411819458008, -3.3527705669403076, 1.2116538286209106, -0.2416006624698639, -0.34136635065078735, -1.157059907913208, -2.802927255630493, -1.9848012924194336, -1.2077419757843018, 1.6508738994598389, -2.8882808685302734, 4.766324520111084, 0.22192060947418213, 1.3552753925323486, 0.5349923372268677, 0.6205495595932007, -1.0053311586380005, 0.7843670845031738, 0.3430745601654053, -1.4240896701812744, 0.6803497076034546, 0.4181385040283203, -3.109506130218506, -0.9680776596069336, 0.0114365816116333, 2.6409549713134766, 1.4787256717681885, -1.3277204036712646, 0.655634880065918, -2.1160969734191895, -1.1864067316055298, 0.3756474256515503, -0.35371434688568115, 0.6393885612487793, -1.849177598953247, 1.931809425354004, 0.17849621176719666, -1.798598051071167, -0.7851386666297913, 4.289490222930908, -0.5763719081878662, 1.0347261428833008, 2.571843147277832, -0.7769291996955872, 2.3897058963775635, 0.05702797323465347, -0.9171275496482849, -0.5687255263328552, -1.572898507118225, -3.007443428039551, -0.8488079309463501, 0.7181192636489868, -0.8131395578384399, -3.2914466857910156, 1.1642653942108154, 0.5088221430778503, -1.2387752532958984, -0.004646033048629761, 1.6687743663787842, 1.1282503604888916, 1.038243055343628, 0.015370473265647888, -1.1450560092926025, 0.10526251792907715, -2.95695161819458, 1.1827419996261597, 0.4647299349308014, -1.2815334796905518, -2.514347553253174, 0.8187432289123535, -0.8467481732368469, 0.8322643041610718, -0.6824021339416504, 2.6241745948791504, -0.9964579343795776, -1.0088515281677246, 2.1802687644958496, 0.25206077098846436, -0.880365788936615, 1.5645233392715454, 1.902915358543396, 2.2546639442443848, -0.6434056758880615, -1.1604217290878296, 1.0899280309677124, 0.4428829848766327, -0.45149821043014526, 1.8185009956359863, 1.1044553518295288, 0.7320768237113953, -1.277651071548462, -0.5446456670761108, 1.367661476135254, 0.4065206050872803, 1.2101880311965942, -1.7825044393539429, -0.971254289150238, 0.028313979506492615, -0.48395299911499023, 0.758901059627533, 0.5180529356002808, 2.3352890014648438, 2.1946887969970703, 2.0854411125183105, 1.9586181640625, -0.3852717876434326, -0.7251476049423218, -0.9404850006103516, -1.6409683227539062, 1.533216953277588, -1.2961159944534302, 1.7623794078826904, 1.1253098249435425, -1.7661480903625488, 0.30013489723205566, 0.21841132640838623, 0.8554924130439758, 0.007741957902908325, -1.1512300968170166, -0.6454878449440002, -0.704291045665741, 0.1926097869873047, -0.4635710120201111, 1.9656662940979004, -1.792330265045166, -2.4902853965759277, -0.9694128632545471, -0.011256694793701172, 0.08572757244110107, 1.3988924026489258, -0.8189443945884705, 0.7938545942306519, 0.11275231838226318, -1.5899274349212646, -3.805279493331909, 0.7592815160751343, 0.5866888761520386, 2.1496434211730957, -0.26841840147972107, -0.8308154344558716, -0.8814694285392761, 1.4472870826721191, -1.3728920221328735, -0.5740512609481812, 1.3805534839630127, -1.0754423141479492, 1.5601407289505005, 0.0542791485786438, -0.4702798128128052, 0.048447661101818085, -0.1453794240951538, -0.13944905996322632, 1.4102944135665894, -2.083555221557617, 2.1314897537231445, 1.6014127731323242, 2.525649070739746, -0.12115776538848877, -2.0609917640686035, 2.4407155513763428, 1.5577216148376465, 0.5321992635726929, 2.8322157859802246, 0.2511873245239258, 1.4323056936264038, -0.5264501571655273, -0.5021815299987793, 1.3651752471923828, -0.36324411630630493, 2.762948751449585, 0.6981598138809204, -2.243684768676758, 1.9745898246765137, 1.403448462486267, 1.3944286108016968, -0.05798834562301636, 0.9823150038719177, 2.8400557041168213, 1.531822681427002, 0.16992753744125366, 0.3071092963218689, 1.9199753999710083, -0.027764201164245605, 2.603733539581299, -0.9969159364700317, -0.7276281714439392, -1.2077720165252686, -1.628577709197998, 0.29938238859176636, 0.2555043697357178, 2.650512218475342, -0.30729779601097107, 0.17833441495895386, -1.7159360647201538, -0.4528810679912567, -0.644244909286499, 0.02955526113510132, -1.5731379985809326, 0.17659330368041992, -0.33294907212257385, -0.7081032991409302, 1.1943038702011108, 1.7470347881317139, 3.0508129596710205, -1.7151317596435547, 1.739090085029602, 0.498685359954834, -1.5451953411102295, -0.39287322759628296, 0.32659295201301575, -0.20925164222717285, 0.35692837834358215, 0.6697062849998474, 2.4450316429138184, 0.24846506118774414, -0.19228693842887878, 0.5567725896835327, -0.4160224199295044, 1.4209462404251099, -0.9346815347671509, -0.8783038258552551, -0.5945990681648254, -1.5196099281311035, 2.227802276611328, -0.8776293396949768, 1.8930836915969849, -0.9980921745300293, 1.011171579360962, 1.7195603847503662, 2.6108086109161377, 1.989898920059204, 0.3744444251060486, -0.4408130645751953, -3.1566786766052246, -0.7937069535255432, -2.7871594429016113, 1.1245977878570557, -1.5307680368423462, 0.22405636310577393, -1.0361032485961914, 2.147557020187378, 2.166611671447754, 0.41436439752578735, 3.1309237480163574, -0.13780555129051208, -0.2658657431602478, 0.7529473900794983, -0.6396013498306274, -2.020772695541382, -1.5245189666748047, -2.649590492248535, 1.3339900970458984, 2.281736373901367, 0.03178203105926514, 0.4388154149055481, 0.5294004082679749, 1.5891401767730713, 2.0755674839019775, -0.7103713750839233, 1.7504701614379883, -2.1545870304107666, 0.7859029769897461, 2.837618112564087, 1.2022819519042969, 1.4688657522201538, -1.5172756910324097, 0.6258145570755005, 0.44819098711013794, -1.0491266250610352, 0.03978189826011658, -0.9326573610305786, 0.8135985732078552, -3.838474750518799, 0.9571987390518188, -4.671871185302734, 0.4946311116218567, 0.6952275633811951, -1.3516136407852173, 1.488096833229065, 0.5377113819122314, -0.3188250958919525, -2.644444465637207, 0.7848604917526245, 1.447829008102417, -1.3926278352737427, -0.4617948532104492, 1.3596150875091553, -0.6487281322479248, 3.4309616088867188, -1.3127630949020386, -0.4603632688522339, 1.7175321578979492, -1.3643672466278076, -0.3642866015434265, 1.838410496711731, 1.750298023223877, -0.5783153176307678, 0.8675613403320312, 0.259804904460907, -2.5139760971069336, -2.4307894706726074, 2.6671299934387207, -0.08640512824058533, 1.17792809009552, -1.7119777202606201, 1.7760924100875854, 0.8011585474014282, 1.5166728496551514, -0.70496666431427, 1.4435179233551025, -0.744814395904541, 1.3611561059951782, 0.034059975296258926, 2.1173338890075684, -2.3506646156311035, -0.4873420000076294, -1.0457569360733032, 1.7556804418563843, 1.4532275199890137, 2.367234945297241, 1.131615161895752, -0.7309475541114807, 2.9883503913879395, 2.493647336959839, -1.6616482734680176, 1.3931337594985962, 2.7374887466430664, 2.6534671783447266, 0.6260320544242859, 1.0267505645751953, 0.5348271727561951, 2.165757656097412, 0.02622470259666443, 1.5205248594284058, 0.5749154686927795, 2.61781644821167, -1.2218221426010132, -1.5520963668823242, -0.0031980425119400024, 0.12765109539031982, -0.7980062961578369, -0.5470058917999268, -0.4350285530090332, 1.0977706909179688, 1.3161848783493042, -0.17789965867996216, 1.4386941194534302, 4.608296871185303, -1.2956860065460205, 0.5792473554611206, -0.5680874586105347, -0.35505858063697815, 0.5308308601379395, 1.4561727046966553, -0.5863896608352661, 1.543924331665039, -1.721414566040039, 1.3792083263397217, 0.8907133340835571, -1.370602011680603, -1.2238082885742188, -0.9522126317024231, -0.03336048126220703, 0.23728680610656738, -0.21426114439964294, -1.5867185592651367, 1.717694640159607, 0.5115398168563843, -1.4194468259811401, -0.8885698914527893, -0.3259825110435486, -1.5684306621551514, -0.8679811358451843, 0.6980901956558228, -0.7231000661849976, 0.2932386100292206, -1.181039571762085, 0.17466236650943756, -0.3979155421257019, -2.559974431991577, -0.883616030216217, -0.31932881474494934, 0.7004626393318176, 0.06477522850036621, 0.9069069623947144, -0.8546804785728455, -0.1100488007068634, -0.9144212603569031, -0.2952275276184082, -1.1675269603729248, -2.647162437438965, -0.05460156500339508, -0.730786919593811, 1.7449944019317627, 2.350515365600586, 0.3739001750946045, 0.9324982166290283, 0.6193861961364746, 3.1230151653289795, -0.8530005216598511, -1.5514881610870361, -0.21190696954727173, -2.4924676418304443, -2.1311299800872803, -1.7353498935699463, 0.2763831615447998, 1.2593507766723633, 0.44011828303337097, 0.39693596959114075, 1.610844373703003, 0.8293320536613464, 2.09043025970459, 0.7119112610816956, -1.6699416637420654, -0.6853539943695068, -2.6074914932250977, 0.15487542748451233, 1.9268524646759033, -0.38744640350341797, 1.4939647912979126, -2.135369062423706, -2.411383628845215, 2.575162172317505, -1.8304861783981323, -1.7500269412994385, -0.9304447174072266, -0.394126832485199, 1.2142739295959473, 1.2369751930236816, -1.4282011985778809, -2.871272087097168, -0.7112314105033875, 0.24478840827941895, 0.5834643244743347, -0.39312413334846497, -1.2905267477035522, 0.6985948085784912, -3.9942595958709717, 1.8949038982391357, -0.11488932371139526, 1.9391875267028809, 1.8914687633514404, -0.07145267724990845, -0.3647243082523346, -2.234077215194702, -1.8928046226501465, -0.9165323972702026, -1.7846260070800781, -1.4224069118499756, -0.9896007776260376, 1.7211198806762695, 1.2272247076034546, -0.10151742398738861, -0.529995858669281, 1.0129177570343018, 1.5865600109100342, -1.8824126720428467, 1.3754053115844727, 1.2717365026474, 0.4585690498352051, 0.07112044095993042, 0.8176493644714355, -0.8773079514503479, -0.33163344860076904, -1.834047794342041, 0.20219333469867706, 0.11304625868797302, 2.7022347450256348, 1.210650086402893, 0.9214558601379395, 0.6052064895629883, 1.1482371091842651, -2.4549570083618164, 1.6657174825668335, 0.014654338359832764, -0.9798338413238525, -2.853675365447998, -0.9313644170761108, -0.9902474880218506, -0.29639798402786255, 0.6956941485404968, 0.8248255848884583, 0.43961992859840393, -0.4088796079158783, -1.9202277660369873, 1.0973680019378662, 1.9680598974227905, -0.4017971158027649, 1.4720640182495117, 0.5577303767204285, 3.350097417831421, 3.6371867656707764, 0.323102742433548, -3.0754809379577637, 1.0751633644104004, 1.3883188962936401, -2.697763442993164, -0.1719818264245987, -0.7272034287452698, 0.9039514064788818, 2.111675500869751, 0.1589149385690689, 1.680578351020813, -1.4089319705963135, -3.0841803550720215, 0.5191684365272522, -2.197262763977051, 1.536440372467041, -0.19839662313461304, 1.2948627471923828, 0.8888232111930847, 1.086686372756958, -2.0802412033081055, 2.3691556453704834, -3.8170523643493652, -0.12297692149877548, -3.728858470916748, -0.7724065184593201, -0.07998353242874146, -0.7635337710380554, 0.025669872760772705, -3.22560453414917, -1.0447672605514526, 2.0668976306915283, 1.6149622201919556, 1.128589153289795, 0.22091197967529297, 1.4329274892807007, -1.5585641860961914, -0.9660998582839966, 1.2571238279342651, 1.0694854259490967, -0.5448180437088013, -1.8757506608963013, 0.786607027053833, 1.2917503118515015, 0.9547399282455444, -0.5172122716903687, -1.1906623840332031, 2.4125661849975586, 2.684403896331787, 0.924970269203186, 3.282299518585205, 0.5569773316383362, -0.3739584684371948, -0.8895300626754761, -1.6013027429580688, 2.0153632164001465, 0.8061255216598511, -2.2903518676757812, -0.7788070440292358, 0.6300203204154968, 0.18969565629959106, -0.9450236558914185, -0.4438260793685913, 0.4874109923839569, -0.7964999675750732, 1.2013792991638184, -0.7229083776473999, 1.5910303592681885, 0.9718819856643677, -0.30150318145751953, 2.9777069091796875, -0.654367208480835, 0.6697643995285034, -0.8304802179336548, 1.4023621082305908, 0.37399494647979736, 0.5569185018539429, 0.37598082423210144, -0.011695697903633118, -1.2434425354003906, -1.397876262664795, -3.3259634971618652, -3.8629934787750244, -0.931708812713623, -1.9285645484924316, 0.10697957873344421, -1.1344809532165527, 0.6956108808517456, 0.8048825263977051, 0.44700416922569275, 0.8133697509765625, -1.1514467000961304, 0.10827599465847015, -0.09151077270507812, 1.6380826234817505, -1.8285554647445679, 3.1479907035827637, 1.2717013359069824, 1.1937828063964844, -0.004682660102844238, -0.5093693733215332, -0.591560423374176, -2.393284559249878, -0.9442692995071411, -1.0501503944396973, -0.9178435802459717, 0.559848427772522, -1.8334110975265503, -0.22804316878318787, 0.5123174786567688, -1.3089256286621094, -1.2324246168136597, 1.6157755851745605, 0.6773996353149414, -1.1941393613815308, 0.8686650395393372, 4.811258316040039, -1.448554277420044, -0.3816739022731781, 0.6840158700942993, 0.6227203607559204, -0.012328624725341797, 1.0832345485687256, -1.2225342988967896, 3.1100175380706787, -0.6999891996383667, 0.13411489129066467, -1.0836780071258545, -2.828157901763916, -0.841345489025116, -0.09774601459503174, 0.18981026113033295, 1.678615927696228, -2.767273187637329, -2.3900465965270996, -1.1493912935256958, 1.633918046951294, 0.8580542802810669]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-70.3802719116211, 7.232798099517822, -25.367155075073242, -36.06809616088867, 19.875965118408203, -20.392141342163086, -12.56828498840332, 12.974183082580566, -37.536827087402344, 1.3631764650344849, 5.656952857971191, -62.837249755859375, -9.825178146362305, 4.724490642547607, 16.662109375, 22.700271606445312, -35.111473083496094, 25.558116912841797, 21.127195358276367, -36.60901641845703, -43.72654724121094, -21.259033203125, -31.07050895690918, -21.253347396850586, -50.86921310424805, 8.823373794555664, -21.852474212646484, -36.35997009277344, -5.369457244873047, 38.349365234375, -35.15652084350586, -12.264811515808105, -29.534027099609375, 32.69220733642578, 7.993452072143555, 17.8831729888916, -17.463884353637695, 20.860877990722656, -32.570045471191406, 6.423421859741211, 7.317106246948242, 41.180240631103516, -66.25405883789062, -3.1359288692474365, -0.25356972217559814, 12.99736499786377, 17.87652015686035, 17.647111892700195, 58.56886672973633, 64.73001098632812, 3.69427752494812, 5.547886848449707, -13.68631649017334, -32.65633010864258, 29.31917953491211, -51.46259689331055, 2.8281350135803223, 47.64773178100586, -88.01737976074219, -51.93169403076172, -20.094722747802734, -23.896156311035156, -24.78452491760254, 27.354846954345703, 35.29511642456055, -20.23788833618164, 46.03910446166992, -27.654054641723633, -0.05425136163830757, -29.595748901367188, -14.388078689575195, 28.575706481933594, -62.150238037109375, 6.369791507720947, -23.140092849731445, 10.308680534362793, -0.6028751730918884, -34.39850616455078, 30.516103744506836, 9.990799903869629, -42.4985237121582, 31.099246978759766, 3.90899920463562, -17.173763275146484, -17.109115600585938, 53.28043746948242, 17.30434799194336, -4.936792373657227, -9.469279289245605, -12.70464038848877, 15.394571304321289, -7.334254264831543, 6.285633563995361, -2.1128060817718506, -16.80062484741211, -5.297868728637695, -40.116851806640625, -14.28211498260498, -64.0633544921875, 16.06214141845703, 13.411510467529297, 53.10615539550781, 56.5428466796875, -8.594623565673828, -20.432043075561523, 9.401373863220215, 73.33441925048828, 76.98926544189453, -5.702310562133789, 11.958243370056152, 40.875030517578125, 14.887544631958008, 11.285428047180176, -36.68199157714844, -20.06122589111328, -21.465246200561523, -68.28678131103516, -26.708663940429688, 42.403316497802734, 15.33642578125, 3.6040964126586914, -5.513535976409912, 21.23036766052246, -30.372676849365234, -30.26480484008789, -1.667766809463501, -6.3878278732299805, 5.204978942871094, -20.23504638671875, -34.77852249145508, -9.189064025878906, 20.484731674194336, -23.623367309570312, -3.7534711360931396, 19.967037200927734, -55.73944854736328, -11.412230491638184, -15.602120399475098, -26.78512191772461, 5.659107208251953, 4.728160858154297, 16.725982666015625, -43.36369323730469, 8.952500343322754, -8.847232818603516, -14.660296440124512, 47.17215347290039, -9.950197219848633, -33.66273498535156, 13.26294231414795, -53.18231964111328, 36.49141311645508, 40.039527893066406, 49.4931755065918, 13.641366004943848, 19.978931427001953, 14.470245361328125, -28.424869537353516, 20.45775604248047, -19.620256423950195, -32.23369216918945, -23.752769470214844, -1.3182446956634521, -11.243657112121582, -14.305191993713379, 37.958194732666016, 14.950972557067871, -19.994049072265625, 28.774986267089844, 52.537723541259766, -26.708873748779297, -8.137843132019043, 56.712646484375, 9.488578796386719, -24.558164596557617, 36.437889099121094, 4.752730846405029, -11.55695915222168, 15.795870780944824, 16.31250762939453, -38.53635025024414, -3.560616970062256, -23.987154006958008, 25.873926162719727, -40.546077728271484, -0.2406822144985199, -46.068058013916016, 5.900144100189209, 33.79330825805664, 8.131111145019531, 0.7108593583106995, -0.9548783898353577, 45.628700256347656, -38.716896057128906, 29.238664627075195, -15.604856491088867, -6.9718122482299805, -34.30290985107422, 9.763628959655762, 13.252584457397461, -15.505977630615234, -1.56631600856781, 0.4433584213256836, 55.05660629272461, 12.506779670715332, 74.94673156738281, 11.910124778747559, 63.919456481933594, 12.934142112731934, -0.590596079826355, 0.08992567658424377, -62.79558563232422, 13.098910331726074, 8.198275566101074, 33.964881896972656, 16.74827003479004, 65.68458557128906, 10.939866065979004, 9.814074516296387, 17.94991111755371, -21.254711151123047, -5.267482757568359, 10.253865242004395, 3.7000484466552734, -8.83556079864502, 14.053376197814941, 21.832290649414062, -51.19118881225586, 2.380460739135742, 28.75025177001953, 4.87778377532959, 5.810163974761963, -50.32929992675781, 31.45857048034668, -22.532773971557617, 37.92656326293945, 29.878515243530273, 33.42597198486328, -30.97509765625, 20.299144744873047, 26.686952590942383, -7.379782199859619, -25.790876388549805, -39.967506408691406, 27.703224182128906, 3.740358352661133, -18.676279067993164, -20.13673210144043, 9.142210960388184, -23.03053855895996, -11.122011184692383, -26.584367752075195, 4.148756504058838, -0.5603976845741272, 59.6422233581543, 30.934852600097656, -11.786089897155762, -8.215471267700195, -96.9067153930664, -40.39387512207031, 33.79945373535156, -12.020612716674805, 9.159664154052734, 16.721729278564453, -30.983497619628906, -21.525348663330078, 31.750370025634766, -8.296286582946777, 31.020980834960938, -19.05724334716797, -8.04732608795166, -6.500242233276367, 15.519983291625977, -10.053350448608398, 5.167413711547852, -0.7998074293136597, 3.105567693710327, 10.393346786499023, -7.574861526489258, -22.697965621948242, 31.00923728942871, -4.0808210372924805, 28.11411476135254, 34.348819732666016, 29.519672393798828, -13.411392211914062, -14.05720043182373, -27.6822509765625, 21.06140899658203, 22.504924774169922, 46.85976028442383, -30.22844886779785, -77.16543579101562, 4.413987159729004, 13.732015609741211, -24.888263702392578, -19.438182830810547, 1.065654993057251, -40.49895095825195, 12.244869232177734, -15.794366836547852, 51.607479095458984, 24.790822982788086, 34.857757568359375, -5.837008476257324, 33.0092887878418, 4.264458656311035, 71.31869506835938, -7.910965442657471, -7.086438179016113, -6.191934108734131, -30.881465911865234, -1.3630043268203735, 20.383399963378906, 42.71514129638672, 49.20361328125, -26.6136474609375, -17.629207611083984, -29.367725372314453, -10.784220695495605, 75.773681640625, -8.47509479522705, -3.3477704524993896, -46.0421142578125, -5.202467441558838, 19.912839889526367, -7.623219013214111, 11.142413139343262, 16.729320526123047, -9.619131088256836, -22.685165405273438, 6.614405632019043, 11.399650573730469, 9.940461158752441, -24.453105926513672, -15.426429748535156, 70.54149627685547, -49.3096923828125, 16.473060607910156, 8.391470909118652, 27.280797958374023, -43.65769958496094, -21.275745391845703, 29.29613494873047, -42.893619537353516, -41.64443588256836, -22.288673400878906, 2.8253285884857178, 41.92096710205078, -3.5534164905548096, 8.525169372558594, 29.691062927246094, 45.091400146484375, -28.62908935546875, -18.23367691040039, -26.787172317504883, -14.902812957763672, -20.975679397583008, -63.551143646240234, 36.66480255126953, -2.881132125854492, 11.932920455932617, -33.9072380065918, -10.012185096740723, 17.04534912109375, -3.0461173057556152, -13.088404655456543, 26.349637985229492, -1.0698612928390503, -26.912708282470703, -35.26853942871094, -2.5658819675445557, -12.821261405944824, 12.170575141906738, -20.167905807495117, 20.055784225463867, -3.670975923538208, -52.31468200683594, -12.41014575958252, -17.227561950683594, -9.763961791992188, -5.5032758712768555, -15.012747764587402, 59.76013946533203, -12.524694442749023, -47.72529220581055, -4.37277364730835, -30.772050857543945, 26.299365997314453, 11.856279373168945, 33.56892395019531, -71.75601196289062, -14.622841835021973, -19.144454956054688, -14.676862716674805, -19.116939544677734, -11.395416259765625, -28.97978973388672, 5.27144718170166, -29.057435989379883, 48.117557525634766, -26.284212112426758, 66.34427642822266, -15.246929168701172, -2.4913477897644043, -62.17097473144531, -39.20011901855469, -6.9589009284973145, 35.1208381652832, -13.275513648986816, -3.3836238384246826, 18.525890350341797, 21.124174118041992, -47.35211944580078, 45.507240295410156, 18.222864151000977, 2.4251749515533447, 9.668807983398438, 44.341148376464844, 22.274721145629883, -3.7745814323425293, -43.649356842041016, -11.548254013061523, 3.2199416160583496, -10.626705169677734, 57.159393310546875, 33.84050369262695, -8.456507682800293, 11.849067687988281, -17.968639373779297, 65.10421752929688, 8.311760902404785, 27.337656021118164, 3.9536731243133545, 10.032075881958008, 8.908276557922363, -13.206921577453613, 6.350522994995117, -59.15740966796875, 61.82257843017578, 33.83766555786133, -13.2438383102417, 21.59122085571289, 9.282316207885742, 6.564874649047852, 13.897672653198242, -58.501197814941406, 16.758468627929688, 36.65386199951172, 15.490972518920898, -43.72124099731445, 21.00629997253418, 11.344649314880371, 35.58420944213867, 13.48264217376709, 46.12225341796875, 15.752506256103516, 31.970991134643555, 15.11279582977295, -4.294302463531494, -64.99334716796875, 16.94305992126465, 21.164302825927734, 15.437525749206543, 29.05473518371582, 18.371761322021484, 16.764869689941406, 15.197741508483887, -12.029791831970215, 32.250308990478516, -2.6734917163848877, -21.864097595214844, 43.6202392578125, 28.78858184814453, 17.718984603881836, 12.926568031311035, -39.01202392578125, -8.248311042785645, -22.303640365600586, 18.214258193969727, 6.418310165405273, 22.953702926635742, 8.566162109375, -96.40411376953125, 42.674285888671875, -5.7290754318237305, -22.5737361907959, 37.782169342041016, -14.499473571777344, 37.958229064941406, 16.034500122070312, -19.90850830078125, 0.9796454310417175, 21.82648468017578, 0.12586498260498047, -21.98189353942871, 20.583803176879883, 41.39045715332031, -37.89618682861328, 30.877422332763672, -66.20044708251953, 43.509376525878906, -23.763273239135742, 8.797761917114258, -4.783697128295898, -25.479209899902344, -15.4909029006958, 7.514346122741699, -26.080936431884766, -19.88851547241211, -4.253057479858398, 10.180415153503418, -3.9810731410980225, 9.305557250976562, -42.851566314697266, 11.661635398864746, 13.098328590393066, 65.40296936035156, 18.663982391357422, -16.334331512451172, 13.518526077270508, -16.569198608398438, 54.088321685791016, -42.74365997314453, 57.625179290771484, 28.07244110107422, -3.370394229888916, 53.632572174072266, -14.933891296386719, -10.86475658416748, 8.091093063354492, 14.33554458618164, -43.71216583251953, 10.52292251586914, 12.03774642944336, 25.823740005493164, -8.815495491027832, -32.46683883666992, -17.74854850769043, 66.45653533935547, -14.877790451049805, 40.056182861328125, -26.66324234008789, -7.834569931030273, -62.07120132446289, -2.442833185195923, -49.13480758666992, 19.984073638916016, -39.481868743896484, -10.039324760437012, 6.621580123901367, -38.01414489746094, 44.83877944946289, -22.812808990478516, -22.895788192749023, -36.85032653808594, -47.179786682128906, 17.464614868164062, 3.9095072746276855, 3.604585647583008, -55.349876403808594, -16.970317840576172, -10.957072257995605, -6.636817932128906, -16.244190216064453, -45.099365234375, 1.2172266244888306, 11.736233711242676, 63.79051208496094, 9.187382698059082, -58.47867202758789, 20.564311981201172, -27.79050064086914, -70.15038299560547, -38.828975677490234, 15.474807739257812, -75.38762664794922, -22.995302200317383, -28.41280174255371, 4.094462871551514, -35.450016021728516, -10.894789695739746, 36.7099609375, 3.994250535964966, -11.075935363769531, -50.50990676879883, 57.143898010253906, -3.145069122314453, -44.856895446777344, 43.58110427856445, 16.51006317138672, 38.028995513916016, 25.68597984313965, -52.048866271972656, -13.67729663848877, -10.169463157653809, -62.96882629394531, -13.524713516235352, 49.57025146484375, -2.6023635864257812, 13.232696533203125, 41.3986930847168, 21.564258575439453, 18.978837966918945, -0.2617461383342743, -25.9627628326416, -0.6411445736885071, 17.315053939819336, -20.51972007751465, 13.395585060119629, -23.520263671875, 2.6281299591064453, -54.601585388183594, 7.704019546508789, 22.918983459472656, 0.7580059170722961, -10.088190078735352, 33.06989669799805, 9.579939842224121, 6.461026668548584, -22.8467960357666, -15.215929985046387, 31.264789581298828, 68.18657684326172, 30.080297470092773, 20.737457275390625, 43.898475646972656, -28.027240753173828, 36.064823150634766, -17.72681999206543, 9.71423053741455, 15.049283027648926, 18.570125579833984, -38.58584976196289, 17.568445205688477, -25.023271560668945, -6.354560852050781, -35.28697204589844, 36.335968017578125, -17.724796295166016, 20.313549041748047, 15.17285442352295, 4.09975004196167, 35.73069763183594, 16.84258270263672, -32.98796844482422, -27.177804946899414, 15.207162857055664, -5.527792930603027, 24.21223258972168, 32.73431396484375, -1.8940809965133667, 3.24316668510437, 36.559226989746094, -53.588783264160156, 2.8514604568481445, -44.85820388793945, -11.120570182800293, -1.1275900602340698, 36.65480041503906, 18.72016716003418, -0.15748330950737, 10.568522453308105, -19.643157958984375, -56.24479675292969, 27.994600296020508, 19.14350700378418, -11.347838401794434, -9.760457038879395, 12.569766998291016, 2.06742000579834, -19.109329223632812, 38.50216293334961, -2.855492353439331, 17.09252166748047, -27.280269622802734, 23.25263786315918, -7.119800090789795, 12.471826553344727, -63.749755859375, -8.93785285949707, 4.646511554718018, -0.819411039352417, 7.916637420654297, 1.4588136672973633, -14.11816120147705, -1.997462272644043, 5.348083019256592, 41.32222366333008, 29.210914611816406, -22.580793380737305, 12.756757736206055, -40.46127700805664, -8.01214599609375, -21.776988983154297, -4.382989883422852, -41.16208267211914, 0.4277839660644531, -0.031247081235051155, 53.4982795715332, 2.613290309906006, -13.12031078338623, 45.5847053527832, 25.446792602539062, 30.97746467590332, -25.411209106445312, 24.271595001220703, -4.263606071472168, 32.61952209472656, -48.667598724365234, -33.96763229370117, -17.676307678222656, 11.467780113220215, -22.998254776000977, -15.987680435180664, -54.31679153442383, 35.162967681884766, 14.804136276245117, 41.64677810668945, 35.72677993774414, -32.39256286621094, -8.947332382202148, 1.1641807556152344, 19.2534122467041, -32.7798957824707, -27.903276443481445, 11.720815658569336, -40.531436920166016, -36.69691848754883, -53.341796875, 23.933469772338867, -21.652013778686523, 13.551715850830078, -3.8769521713256836, 56.85763168334961, -40.59082794189453, -21.1304988861084, 2.8851263523101807, 2.855945348739624, -15.148435592651367, 22.298994064331055, -10.682319641113281, 65.03776550292969, 3.9406492710113525, 2.3803653717041016, 24.087932586669922, 9.080495834350586, -19.28097915649414, 17.722211837768555, -5.231178283691406, 57.946311950683594, 19.324975967407227, -23.606428146362305, -10.39951229095459, -11.789385795593262, 8.866774559020996, -23.00998878479004, 40.18157196044922, -24.26432991027832, 7.649386405944824, 24.531965255737305, -51.22087097167969, -28.483579635620117, 15.140270233154297, 29.493494033813477, 9.840865135192871, -38.163509368896484, 12.897571563720703, -25.327119827270508, -13.644543647766113, -0.4251554012298584, -75.57122802734375, -26.465919494628906, -27.871456146240234, -2.1134283542633057, -4.393586158752441, -28.497915267944336, 29.419849395751953, 11.791460037231445, -27.42702293395996, 22.116769790649414, 52.322444915771484, -55.69298553466797, -3.978346824645996, 59.299102783203125, -19.54294204711914, -11.043413162231445, 30.57067108154297, 5.196438312530518, -32.221824645996094, -9.795724868774414, -33.10469055175781, -25.329082489013672, -8.136970520019531, 11.00175952911377, 25.77073860168457, -38.27239227294922, -29.959949493408203, 24.189340591430664, -3.5226893424987793, 31.944133758544922, -43.29291534423828, 4.596971035003662, -10.9348783493042, 34.6253776550293, -5.746763229370117, 15.789454460144043, 10.965130805969238, -57.57318878173828, -28.323657989501953, -13.552054405212402, 24.82427978515625, -18.564624786376953, 7.255795955657959, -13.49316120147705, 22.757692337036133, -0.6531633734703064, 18.239843368530273, 32.383689880371094, -32.41630172729492, 19.972444534301758, 4.818636894226074, -3.817728042602539, 0.7210376262664795, 27.590065002441406, 3.2706758975982666, -18.450220108032227, 1.2536847591400146, -45.21648025512695, -49.130828857421875, 27.574085235595703, -0.2495380938053131, 31.82817268371582, -13.361525535583496, -23.973989486694336, -40.69231414794922, -2.6881954669952393, -12.6725492477417, 0.34570735692977905, -30.934009552001953, 35.38825225830078, -48.82893371582031, -1.3757258653640747, -25.374399185180664, 30.244539260864258, 35.17527770996094, 14.77575969696045, 39.991275787353516, -26.651288986206055, -18.75825309753418, 47.58678436279297, 1.0333930253982544, 26.23699188232422, -42.25688552856445, -42.29110336303711, 28.521886825561523, 17.530237197875977, 8.14139461517334, -12.388265609741211, 12.872418403625488, -4.954996585845947, 13.510957717895508, -25.40485382080078, 25.21697998046875, 34.253414154052734, -2.142418146133423, -14.315332412719727, 1.207892656326294, 28.22542953491211, -23.223791122436523, 41.17087173461914, 14.655157089233398, -19.722078323364258, 9.046218872070312, 72.12498474121094, -39.13025665283203, 31.626327514648438, 41.15711212158203, 1.2008013725280762, 2.882272481918335, -63.14250564575195, -47.180973052978516, 22.487998962402344, -6.22028112411499, -20.226531982421875, 5.409943103790283, 55.72205352783203, 10.408580780029297, -11.079345703125, -7.484489440917969, 17.33208656311035, 20.981698989868164, -12.391886711120605, -28.902700424194336, -2.119448661804199, 30.79920196533203, 37.473819732666016, -13.026572227478027, -10.03820514678955, 59.65650939941406, 24.76691436767578, 34.31572341918945, -29.91156005859375, 1.835957646369934, -51.120025634765625, -5.970566272735596, -10.598528861999512, 7.185164451599121, 22.562597274780273, 49.002586364746094, -70.32767486572266, -43.1022834777832, -52.01904296875, -46.773677825927734, 33.700897216796875, 49.229225158691406, 27.101261138916016, -38.30957794189453, 2.432539224624634, 32.999271392822266, -68.32740020751953, -53.7802848815918, -31.423015594482422, -0.520723283290863, -38.20705795288086, -42.94361877441406, 37.293277740478516, -19.60587501525879, 3.125098466873169, -4.502986431121826, 25.76126480102539, 27.36354637145996, 38.9455451965332, -28.364513397216797, -14.935670852661133, 3.953005790710449, 55.87860870361328, 6.446824550628662, 21.789356231689453, 34.01350784301758, 27.143661499023438, -11.427424430847168, 25.17696762084961, 41.88334655761719, -28.341445922851562, 24.161985397338867, 28.66339111328125, -16.46602439880371, 14.975298881530762, -3.1504008769989014, -66.98683166503906, 18.863136291503906, 12.269170761108398, 29.181968688964844, -21.67084312438965, -24.2864933013916, 8.274360656738281, -22.4439697265625, -79.21369171142578, 25.043691635131836, -52.95989990234375, 31.475215911865234, 73.1767578125, 25.472179412841797, 16.05143165588379, -7.242363929748535, -13.531681060791016, -33.510406494140625, -26.76458740234375, -29.37896728515625, -5.320862770080566, -5.261324882507324, 3.768949031829834, 4.218991756439209, 39.34283447265625, 32.8668212890625, 14.447691917419434, 9.891127586364746, -83.68338012695312, 2.9257566928863525, -29.97710418701172, -19.824012756347656, -32.5056266784668, -15.0140962600708, 17.33592414855957, -1.191444754600525, -26.662616729736328, -18.65496253967285, -14.178478240966797, -49.599857330322266, 0.08007215708494186, 31.679698944091797, -38.765506744384766, 18.45718765258789, -0.4047032296657562, 52.01816177368164, -32.46415710449219, -17.949119567871094, 18.488073348999023, -1.735150933265686, 33.148529052734375, 55.91305160522461, -28.089916229248047, 40.875732421875, 4.2849578857421875, 7.756077766418457, -7.124688625335693, 2.5571913719177246, 19.360597610473633, -21.115894317626953, -5.992153644561768, 14.14814281463623, -9.66111946105957, -14.439208984375, 22.761266708374023, 58.04347610473633, 10.830440521240234, 24.008148193359375, 4.872424125671387, 13.729028701782227, 61.859928131103516, -9.850054740905762, 7.068334579467773, -7.042683124542236, 52.23188400268555, 35.944862365722656, -34.72288513183594, -16.57920265197754, 1.8692853450775146, -8.80463981628418, 15.842428207397461, -9.66860580444336, -74.28893280029297, -1.39750337600708, -36.11276626586914, 7.384024620056152, 32.177120208740234, 34.950687408447266, -44.950469970703125, 29.681316375732422, -15.31790828704834, 20.029356002807617, -1.1238383054733276, 41.6215705871582, -24.78512954711914, 37.985206604003906, 15.021430969238281, 42.79158401489258, -14.57548999786377, -63.686805725097656, -26.680252075195312, -42.206756591796875, 15.63279914855957, 39.18526840209961, 16.240234375, -46.510032653808594, 54.90080261230469, -38.77179718017578, 51.84281921386719, 8.321074485778809, 6.816561698913574, -17.134096145629883, -5.011202812194824, 21.24243927001953, -6.723714351654053, -5.471972465515137, -5.0587592124938965, 54.95174026489258, 58.7422981262207, -42.07777404785156, -15.372721672058105, 13.050202369689941, 36.18404769897461, 73.2926025390625, -19.015918731689453, -2.6167731285095215, -6.000104904174805, 38.48717498779297, -66.01021575927734, -44.53679275512695, 13.301505088806152, 46.13138198852539, 4.911872863769531, 8.5123872756958, 17.93027114868164, 13.168594360351562, -7.951107978820801, -10.595115661621094, 12.136359214782715, 25.936594009399414, 16.51758575439453, -10.453028678894043, 14.452910423278809, 22.28337287902832, -11.68525218963623, 30.860431671142578, 12.686542510986328, -1.4139389991760254, 17.043413162231445, -18.96160888671875, -39.939422607421875, 26.334640502929688, -60.80475616455078, -31.922300338745117, -11.797866821289062, -23.873369216918945, 7.823724746704102, -10.355888366699219, 32.065914154052734, -40.491695404052734, 13.900327682495117, -27.699892044067383, -20.170181274414062, -3.1030995845794678, -44.102142333984375, 22.76345443725586, 8.166452407836914, 34.41797637939453, 37.7388916015625, 7.415640830993652, 21.876440048217773, -44.195011138916016, -45.577999114990234, -21.56243133544922, 46.56098175048828, -67.98238372802734, 42.202117919921875, -25.29929542541504, -0.7362054586410522, 24.324880599975586, 37.26850891113281, -15.958564758300781, -20.600107192993164, -25.831769943237305, 87.1538314819336, 22.78240966796875, -53.40465545654297, 9.762901306152344, 41.08916091918945, 18.523042678833008, -37.37118911743164, -59.21418762207031, 0.28638747334480286, 23.585987091064453, -6.666337490081787, 6.6480255126953125, 6.728484630584717, 4.263582229614258, 24.419246673583984, -65.3057632446289, 13.018477439880371, 23.177635192871094, -20.407424926757812, -14.044490814208984, 57.54286193847656, -13.696417808532715, 36.39586639404297, -12.14013385772705, 23.428014755249023, 63.810096740722656, 21.52679443359375, -6.472898960113525, -12.476381301879883, 43.41316223144531, 15.924280166625977, 22.45111846923828, 72.71207427978516, 55.05112075805664, 6.976860046386719, 22.015125274658203, 4.0269775390625, 19.057401657104492, 38.18840408325195, -12.232402801513672, 25.4998836517334, 17.58125877380371, 2.8412582874298096, 20.366378784179688, 44.782508850097656, 37.822322845458984, -25.6546630859375, -0.5896685719490051, -2.6506905555725098, -5.166535377502441, -12.932792663574219, -53.46446990966797, 5.857086181640625, -0.7355448007583618, -15.891960144042969, 9.139848709106445, 13.199664115905762, -3.6866323947906494, 64.66470336914062, 2.485408067703247, 12.06801986694336, -8.475680351257324, -3.1597957611083984, -35.06295394897461, -1.8567118644714355, -61.855743408203125, 74.63119506835938, 17.955364227294922, -14.890525817871094, -21.66864776611328, -12.169989585876465, -10.741521835327148, 6.196163177490234, -13.213842391967773, -31.81523323059082, -16.7216739654541, 38.626495361328125, 36.55460739135742, 2.527153968811035, -35.58537292480469, -15.651824951171875, -18.58568000793457, -17.37241554260254, 4.966081142425537, 7.061161041259766, 4.000761032104492, -1.3440276384353638, 15.862112045288086, 23.494897842407227, -18.95922088623047, -26.664081573486328, 17.345600128173828, 34.83110046386719, 13.246453285217285, -0.4606172442436218, 28.023189544677734, -10.238395690917969, 40.39520263671875, 10.094334602355957, -11.613014221191406, -39.567054748535156, -76.84795379638672, 3.73968768119812, -6.8541059494018555, 28.216205596923828, 12.481620788574219, 26.453689575195312, 16.37893295288086, 47.78592300415039, -22.024105072021484, -23.621673583984375, 7.517452239990234, -12.474690437316895, -43.34346008300781, -2.4672133922576904, -35.7599983215332, 0.8717297911643982, 47.32576370239258, -37.25407791137695, 28.78774642944336, 17.21872901916504, -25.104228973388672, -14.8321533203125, -40.191810607910156, 10.4482421875, 18.294166564941406, -32.005130767822266, 16.393848419189453, 6.472519874572754, 29.543230056762695, 6.582925796508789, -17.472408294677734, -11.021917343139648, 11.9365234375, -9.914613723754883, 2.9754257202148438, -23.5567626953125, 25.138628005981445, 6.304671764373779, 37.625118255615234, -21.493799209594727, -28.411964416503906, 34.72181701660156, -3.033951997756958, -8.806699752807617, 13.279268264770508, -3.8367998600006104, 34.46504592895508, 59.38164520263672, 9.740596771240234, -26.513874053955078, 5.689810276031494, 11.55591869354248, -24.865795135498047, 11.907506942749023, 61.97413635253906, 34.98604965209961, 0.3861178159713745, -2.4271533489227295, -11.62835693359375, -41.63849639892578, -35.798641204833984, 19.511259078979492, -8.462089538574219, 44.727439880371094, 13.603754043579102, -13.082491874694824, -4.580745697021484, -11.924616813659668, -46.22380065917969, 13.997483253479004, -1.8547786474227905, 59.40962600708008, -65.36204528808594, -21.335411071777344, 29.040939331054688, -7.437338829040527, -0.15990500152111053, 32.008705139160156, -11.997532844543457, 28.838077545166016, 20.831924438476562, 35.54295349121094, -45.33955001831055, 33.86312484741211, -5.630966663360596, 0.8981621265411377, -31.6639461517334, 9.016738891601562, 17.549243927001953, 41.61471176147461, -36.76666259765625, 37.820003509521484, -3.4538841247558594, 4.909021377563477, -56.23451232910156, 37.338016510009766, -26.965484619140625, -3.5519556999206543, 7.468518257141113, -5.114766597747803, -40.18703842163086, 27.59132194519043, -32.164974212646484, -4.905284881591797, -67.32577514648438, 7.831831455230713, 47.60266876220703, 20.468948364257812, -10.215994834899902, -31.056392669677734, -7.040217399597168, 4.579013824462891, -36.289100646972656, -56.429691314697266, -32.46049499511719, 58.4326286315918, 35.33042526245117, 25.69598960876465, 13.053828239440918, -18.063261032104492, -13.477514266967773, 31.913551330566406, -23.8516845703125, 9.49644947052002, 78.75096893310547, 1.9664499759674072, 36.202613830566406, 18.354707717895508, 47.49375915527344, -18.796459197998047, 9.023639678955078, -69.74608612060547, 0.43972331285476685, -0.6599271297454834, 4.339449405670166, 0.6791276931762695, 14.54510498046875, 14.104434967041016, 97.22756958007812, -36.79775619506836, 17.114093780517578, -37.03422927856445, -14.061331748962402, 17.993389129638672, 1.3206169605255127, 9.20505142211914, 14.18150806427002, 9.597228050231934, -31.63068389892578, -3.1953189373016357, 60.647613525390625, 18.492877960205078, -42.65629196166992, 44.45500564575195, 37.86103439331055, 6.314180850982666, 6.41828727722168, 16.034887313842773, 24.82162094116211, 32.00081253051758, 31.47442054748535, -39.86452102661133, -27.395584106445312, 54.54508972167969, 29.77239418029785, -22.70932960510254, 58.77306365966797, 41.503028869628906, 3.2716314792633057, 34.06111145019531, 34.99065017700195, -13.889447212219238, 32.25169372558594, -6.43330192565918, 18.568897247314453, 28.678009033203125, 35.34973907470703, -13.572898864746094, 13.886130332946777, 28.38375473022461, -41.66321563720703, 34.27347946166992, 55.485260009765625, -30.92789077758789, 42.41036605834961, 42.0914421081543, -30.980684280395508, -0.33409661054611206, -59.062278747558594, -16.575414657592773, -44.982139587402344, 59.68044662475586, -26.722925186157227, -24.340576171875, 13.461453437805176, 39.40870666503906, -59.123050689697266, -10.72458553314209, 28.754528045654297, 6.77980375289917, 9.80062484741211, 18.37993621826172, 1.1716821193695068, 56.933555603027344, 19.563892364501953, -12.29982852935791, -66.96664428710938, -6.759334087371826, 10.324175834655762, -40.35903549194336, -15.799310684204102, -42.54575729370117, 42.4838981628418, -40.53985595703125, 43.577171325683594, -10.89166259765625, 29.493270874023438, 10.030939102172852, -19.169254302978516, 15.741744995117188, 0.5465199947357178, -16.1525821685791, -21.724464416503906, -31.614765167236328, -47.274837493896484, -26.708087921142578, -12.891871452331543, -13.82163143157959, -44.0605354309082, 2.2417819499969482, 0.8555938601493835, -43.88106155395508, -15.94719123840332, 8.609350204467773, -9.08770751953125, 6.5088791847229, -13.929332733154297, -16.132972717285156, -49.101314544677734, -0.6196983456611633, -0.07696440070867538, -10.85860824584961, 54.221435546875, 56.03355407714844, -45.03101348876953, -1.1889920234680176, -28.96978759765625, -43.34142303466797, 59.96811294555664, -33.724517822265625, 39.72711944580078, 19.526790618896484, 18.827468872070312, -16.59623908996582, -18.956628799438477, 18.18912696838379, 12.002398490905762, -20.182512283325195, 5.341672897338867, -0.7973251342773438, -10.414800643920898, -40.97328186035156, -7.623233318328857, -5.14642858505249, -13.377074241638184, 14.392666816711426, -14.592140197753906, -2.6364452838897705, 28.06542205810547, -25.526819229125977, -12.586112976074219, -31.87034797668457, 6.676499843597412, 23.835269927978516, -34.675228118896484, 8.134202003479004, 4.6036152839660645, -31.392087936401367, -13.084430694580078, -82.3721923828125, 76.12445068359375, 13.914997100830078, 40.57964324951172, -6.664553165435791, -4.684622764587402, 38.696651458740234, -25.37155532836914, 21.23723793029785, -5.886996746063232, 7.1921234130859375, 0.7389582395553589, -0.4531160593032837, -8.475367546081543, 80.49374389648438, 34.283756256103516, 59.07068634033203, 13.704192161560059, 1.6200650930404663, -53.750221252441406, -2.9766032695770264, -23.584104537963867, -1.4973220825195312, -11.239028930664062, 71.68197631835938, -42.20736312866211, 5.449264049530029, -33.566162109375, -29.850502014160156, -23.255821228027344, 11.960916519165039, 10.346794128417969, -44.35529708862305, 42.76275634765625, -12.657729148864746, -8.7356538772583, -23.423425674438477, -3.3203723430633545, 16.143604278564453, 55.15365982055664, -30.52113151550293, 78.81586456298828, -2.2381861209869385, -47.93315505981445, 1.6232025623321533, 10.208355903625488, -60.68721008300781, -7.50930118560791, 36.17900466918945, 12.958575248718262, 18.798425674438477, 17.30425262451172, -8.554543495178223, -0.6167024970054626, -35.053531646728516, 28.278989791870117, -80.15234375, 0.5355323553085327, -29.4377384185791, -33.20338439941406, -2.0355629920959473, 18.288820266723633, -35.825645446777344, 44.58856201171875, 4.591075897216797, -5.375439643859863, -27.49771499633789, -25.144824981689453, 48.8620719909668, -32.9837532043457, -11.500990867614746, -25.563425064086914, 30.563661575317383, 11.071699142456055, -12.305303573608398, -10.992657661437988, 49.94951629638672, 3.7015879154205322, -37.609622955322266, -23.466014862060547, -12.224906921386719, -54.19478988647461, 27.306875228881836, 19.5314884185791, -22.446697235107422, -12.947078704833984, -8.025806427001953, -12.43270492553711, 51.115875244140625, -10.46022891998291, 5.13071346282959, 25.232694625854492, 19.3815860748291, 37.37993621826172, -18.689558029174805, 10.710628509521484, -27.158029556274414, 35.634342193603516, -22.733882904052734, 42.979530334472656, -25.15996742248535, -16.65728187561035, -17.174528121948242, -35.99703598022461, 66.42049407958984, -8.850855827331543, -24.46923828125, -36.41115188598633, -11.353278160095215, 49.839942932128906, 49.086830139160156, 9.198481559753418, 31.967971801757812, -3.3982532024383545, 14.036648750305176, 0.47647637128829956, 21.21283531188965, 51.60072326660156, -31.822275161743164, 26.595430374145508, 12.88826847076416, -6.031828880310059, -8.361832618713379, 45.157928466796875, 26.640308380126953, 36.788818359375, 14.239484786987305, -38.7940673828125, 15.505693435668945, -24.021753311157227, 10.280841827392578, -5.4273362159729, 57.85406494140625, 11.901187896728516, -36.71696472167969, 0.29979392886161804, -0.7842405438423157, 43.65563201904297, 0.9703373908996582, -17.403743743896484, 24.728601455688477, 48.36496353149414, 1.00064218044281, 25.1370849609375, -31.495330810546875, 2.213655471801758, -40.22034454345703, -13.635982513427734, 26.57840919494629, 8.315434455871582, 12.953960418701172, -4.746237754821777, -39.43748474121094, -23.383275985717773, 15.883234024047852, 4.075048446655273, 33.72355270385742, -31.871837615966797, -25.667497634887695, -18.91233253479004, 12.957419395446777, 33.58154296875, 33.06462860107422, 0.9581485390663147, -9.30981731414795, 25.318195343017578, 11.282139778137207, 46.64248275756836, -24.41180419921875, 5.244281768798828, 44.27828598022461, 9.94874095916748, -5.399929046630859, -25.93467140197754, 0.34645143151283264, 6.997529029846191, -20.50389862060547, 21.53661346435547, 8.789789199829102, -48.95093536376953, -20.160547256469727, -47.30329132080078, -25.212053298950195, -3.811697006225586, -30.603504180908203, 10.181665420532227, -88.21048736572266, -25.840499877929688, 36.22838592529297, -10.646245956420898, -20.5072078704834, -10.307576179504395, -5.115795135498047, -10.78196907043457, -16.777311325073242, -54.07036209106445, -33.248783111572266, -40.199920654296875, -17.339130401611328, 20.150447845458984, 18.235519409179688, 5.783517837524414, -22.812952041625977, -5.194029331207275, -13.464971542358398, 5.514763355255127, 11.475584983825684, 23.782466888427734, -5.130748748779297, 8.709391593933105, 48.40797424316406, 17.751220703125, -82.335693359375, 78.16664123535156, 47.6627197265625, -7.058732986450195, -59.35983657836914, -32.53873062133789, 15.667680740356445, -1.1127995252609253, 21.996578216552734, 31.3991756439209, 6.1844587326049805, -32.34490966796875, 0.8028006553649902, -15.98867130279541, 25.450984954833984, -17.400020599365234, 22.534488677978516, 55.6357421875, -0.6308506727218628, 55.266353607177734, 33.40507125854492, -27.386940002441406, -26.812358856201172, 11.653940200805664, -11.310461044311523, 22.45195960998535, -47.59206771850586, 31.91585922241211, -13.791780471801758, -3.380664825439453, 13.997845649719238, 10.53586196899414, -39.29772186279297, 47.53271484375, 28.83570098876953, 27.570972442626953, -12.730692863464355, 29.24619483947754, 22.703866958618164, 24.86159896850586, 28.709518432617188, -9.431378364562988, 22.422515869140625, -35.531227111816406, -61.09695053100586, -62.3505744934082, -3.5658843517303467, -23.290903091430664, -5.127872467041016, 2.874276876449585, -14.448861122131348, 27.753963470458984, 9.011594772338867, 10.038185119628906, 53.729129791259766, 12.747065544128418, -55.11207962036133, -42.81727600097656, -22.969398498535156, -24.630390167236328, -23.174713134765625, 24.892059326171875, -15.151278495788574, -2.5663669109344482, -14.597137451171875, 54.18644714355469, 11.851815223693848, -16.46470069885254, -7.030853748321533, -40.196998596191406, -42.61616897583008, -14.989603042602539, 15.564872741699219, 7.805861949920654, 43.2641716003418, 20.855209350585938, 14.128232955932617, -34.034156799316406, -40.15093231201172, 11.006547927856445, 16.609535217285156, -4.54167366027832, -25.338638305664062, 48.06466293334961, 4.622076511383057, 9.686885833740234, 57.63676452636719, 23.260581970214844, 14.667684555053711, 24.9731388092041, -30.587907791137695, 24.480377197265625, 8.302386283874512, -13.97397232055664, -27.600692749023438, -19.251096725463867, -23.417797088623047, -25.365009307861328, -67.62065887451172, 45.82984924316406, -9.50864028930664, -15.6197509765625, -1.984856367111206, 24.462583541870117, 5.700321674346924, 39.298458099365234, -21.974018096923828, -2.179049015045166, -22.547067642211914, 53.40574645996094, -15.806259155273438, 47.67843246459961, 33.98647689819336, 13.219535827636719, 22.609689712524414, 46.12163162231445, 28.31455421447754, -57.670230865478516, 8.499711036682129, -57.44157791137695, -8.652853012084961, -0.36651578545570374, 6.725930213928223, 11.8239164352417, -19.2420711517334, 1.5882177352905273, 25.361064910888672, 59.39027404785156, -63.357139587402344, 5.678493022918701, -11.390157699584961, -13.785653114318848, 68.94613647460938, -54.85292434692383, 65.43790435791016, 2.730827808380127, 29.99626922607422, 11.321274757385254, 47.27613830566406, 11.028942108154297, 10.018348693847656, 3.297701835632324, -29.0684757232666, -8.646963119506836, -2.3222196102142334, -30.869251251220703, 36.041175842285156, 65.7213363647461, 17.99671745300293, 38.12257385253906, 14.90026569366455, 44.40232467651367, -33.55497741699219, 24.274261474609375, -18.782615661621094, -13.107205390930176, 11.956217765808105, -26.343082427978516, -16.824316024780273, -37.056251525878906, 9.726585388183594, 10.753036499023438, 0.8589518666267395, 5.640254974365234, 8.804018020629883, 47.199893951416016, -3.845829486846924, 2.0317721366882324, 41.35263442993164, 39.750823974609375, -66.2657241821289, 6.067073345184326, -15.228256225585938, -7.26841402053833, -47.27743148803711, 17.964818954467773, -55.79731750488281, -21.667966842651367, -22.381216049194336, 1.7092444896697998, -34.42591094970703, 6.961832523345947, -62.018009185791016, 37.92774963378906, -20.05229377746582, 45.831092834472656, -21.87137222290039, 23.85355567932129, -15.935944557189941, -20.430965423583984, 7.810383319854736, 7.23194694519043, -6.449059963226318, 22.180377960205078, 11.194168090820312, -17.31592559814453, -44.82404708862305, -9.953997611999512, -17.376907348632812, 7.10992431640625, 10.170615196228027, -19.932092666625977, -18.873567581176758, -15.677163124084473, 11.96114444732666, -16.248050689697266, 16.894289016723633, -4.773044109344482, -22.021516799926758, -23.41036033630371, -36.83962631225586, 47.724761962890625, 12.284131050109863, -11.555390357971191, 31.088722229003906, 49.47072982788086, -32.3758430480957, 22.276247024536133, -20.59356117248535, -50.91341018676758, -72.37327575683594, 4.9474968910217285, -31.029403686523438, -21.934131622314453, -20.142566680908203, 9.007433891296387, -16.075305938720703, -25.106534957885742, 29.472126007080078, -12.623268127441406, -23.826080322265625, -4.269424915313721, -18.284555435180664, -21.206796646118164, -30.240150451660156, -0.9676370620727539, -46.39805221557617, -19.738210678100586, 31.631322860717773, -4.922642707824707, 34.6399040222168, -42.78402328491211, 9.749655723571777, 25.28092384338379, 51.906944274902344, -41.773895263671875, -27.535442352294922, -36.44843673706055, -25.175413131713867, 13.369662284851074, -5.376947402954102, -29.925025939941406, 57.05321502685547, 39.1214714050293, 9.49188232421875, -39.24190139770508, 12.701591491699219, 30.3504638671875, -14.213072776794434, -43.865631103515625, -48.49229049682617, -21.730369567871094, -47.95846176147461, 20.43290138244629, -13.57483959197998, -7.227297782897949, 0.16698570549488068, 0.6589246392250061, -0.8029267191886902, 53.605167388916016, -19.151649475097656, 4.4362640380859375, -13.071226119995117, -24.674707412719727, -45.1633415222168, -30.804847717285156, 0.026614036411046982, 7.8176655769348145, -38.86742401123047, -29.691518783569336, -7.39115571975708, 30.969589233398438, -15.587564468383789, 4.572359085083008, 21.585575103759766, -12.227629661560059, -44.747215270996094, 4.214343547821045, 20.406394958496094, 14.957724571228027, -32.430335998535156, 11.341550827026367, 1.5538150072097778, -7.890927314758301, 20.855798721313477, -4.412883758544922, 40.07646942138672, 19.67609405517578, -27.365480422973633, -5.173284530639648, -28.82573699951172, 1.4567476511001587, -13.558576583862305, 26.909406661987305, -32.708919525146484, 23.852821350097656, 31.52776527404785, -3.7153122425079346, -41.49257278442383, 3.2183237075805664, -14.256969451904297, 7.383596420288086, -2.580732583999634, -11.901641845703125, -0.797748863697052, -12.075243949890137, 31.002681732177734, 1.576648235321045, -53.035743713378906, -20.533479690551758, 15.285195350646973, 35.13597869873047, 39.36547088623047, -54.899200439453125, -15.639992713928223, 25.681180953979492, -27.46664047241211, -19.157264709472656, 7.435921669006348, -5.954522132873535, 26.634809494018555, -12.143803596496582, 41.41385269165039, 4.203954219818115, -42.79373550415039, -2.875624418258667, -47.33832550048828, -4.085371971130371, -3.555614471435547, 16.563982009887695, 11.154681205749512, -7.29192590713501, -9.748165130615234, 63.95744705200195, 34.49067306518555, -41.76256561279297, -0.19512152671813965, -15.119284629821777, 44.79707717895508, -5.351367473602295, 9.882917404174805, 10.680414199829102, 57.344234466552734, 46.01564025878906, 12.79689884185791, -63.11722183227539, 0.3164198100566864, 33.402191162109375, 13.866491317749023, 5.713396072387695, 18.420307159423828, 49.441062927246094, -22.45465850830078, 39.65354537963867, -57.06610870361328, -17.4935245513916, -16.685279846191406, 4.326619625091553, -25.756677627563477, 8.293581008911133, 2.6288373470306396, -21.158964157104492, 23.672021865844727, 25.89851951599121, 5.403981685638428, 27.88136863708496, -25.301185607910156, 16.65070915222168, 26.477825164794922, -10.367362022399902, -16.57927894592285, 21.444358825683594, 56.09403991699219, 4.033736705780029, 27.177837371826172, 40.08644104003906, 27.177488327026367, -24.364681243896484, 43.000179290771484, -7.490283489227295, -4.691511631011963, 23.85893440246582, -0.38624143600463867, 38.23524856567383, -1.0888617038726807, -38.30677795410156, -17.544918060302734, 29.364980697631836, -17.71578025817871, 20.40672492980957, -33.054317474365234, 15.98642349243164, 5.011481761932373, -12.078960418701172, -24.203655242919922, 18.456607818603516, -25.47407341003418, 32.56424331665039, -13.418651580810547, 45.843692779541016, -14.98107624053955, 9.345057487487793, 71.57086181640625, -0.7751492857933044, 24.2402400970459, 61.814510345458984, -12.164966583251953, -9.507966041564941, -38.6567497253418, -12.8321533203125, -57.41634750366211, 2.9557597637176514, 0.010053299367427826, 1.859911322593689, -42.203697204589844, -12.039579391479492, 19.03289794921875, 7.106139659881592, 16.975486755371094, -0.2971118688583374, -53.821022033691406, -13.25619888305664, -45.085716247558594, 4.750831127166748, -21.782459259033203, 5.204935550689697, 5.321088790893555, -10.4520263671875, 38.535701751708984, 29.489728927612305, -42.29239273071289, 9.554519653320312, -21.245269775390625, -0.5067075490951538, 18.00575065612793, 6.031203746795654, -9.203575134277344, 39.6218147277832, -17.600624084472656, -33.15660858154297, 25.60724639892578, 40.11610412597656, 36.803958892822266, 50.08192443847656, -45.04658126831055, -22.121990203857422, -57.3233528137207, -60.020118713378906, -60.164180755615234, -34.55415344238281, -8.918465614318848, 37.80806350708008, -27.320465087890625, 1.1793713569641113, -36.44627380371094, -0.8949957489967346, 4.780032157897949, -10.521677017211914, -6.582282543182373, -5.091798305511475, -37.72770690917969, -26.933191299438477, -22.432327270507812, 19.200441360473633, 16.797508239746094, -38.837074279785156, 71.60989379882812, -4.74308443069458, 3.397411584854126, -19.019742965698242, 26.590255737304688, -45.62880325317383, -52.95013427734375, -24.843549728393555, -72.1311264038086, 23.005516052246094, 52.91855239868164, 11.04855728149414, 32.18288040161133, 31.019121170043945, -58.35102462768555, 6.995818138122559, -11.015656471252441, 19.784433364868164, 23.08649444580078, 10.457651138305664, -0.8599602580070496, 19.921146392822266, -4.382962226867676, 17.38324737548828, -7.346278667449951, 28.624256134033203, 60.26812744140625, 13.997227668762207, 0.4691984951496124, 30.445375442504883, 44.72824478149414, -103.3645248413086, -12.145696640014648, -2.3861000537872314, -19.535675048828125, 14.073592185974121, -60.168601989746094, 5.79801607131958, -18.00124168395996, -33.080440521240234, 42.74187088012695, -28.66339111328125, -0.31492161750793457, -21.256479263305664, 11.625831604003906, -2.4908761978149414, 9.973764419555664, -4.356865406036377, 46.99885177612305, -6.886096477508545, -20.88016128540039, -68.38764190673828, -18.447233200073242, -24.25855827331543, -1.0849616527557373, 6.974615097045898, -4.626049041748047, 56.8859748840332, 3.5699102878570557, 28.901470184326172, 45.63230895996094, -33.79014205932617, -28.031726837158203, -6.6771650314331055, -38.18827819824219, -0.07960198074579239, 4.665218830108643, -10.788727760314941, 63.766258239746094, 22.552669525146484, -0.9313286542892456, 11.749809265136719, 22.62406349182129, -46.106040954589844, -10.065162658691406, 31.16644287109375, 12.89263916015625, 44.678802490234375, -30.905052185058594, -30.235078811645508, 9.022226333618164, 29.497339248657227, -7.688450336456299, -0.6539162993431091, -43.419883728027344, -3.270393133163452, 36.91284942626953, -19.116844177246094, 20.497385025024414, -0.8907549381256104, -7.560208320617676, -16.698501586914062, 34.64130783081055, 0.11508653312921524, -43.31779861450195, -0.06744424998760223, 43.81008529663086, 4.9987711906433105, 24.9788761138916, 0.7589619159698486, -22.501863479614258, -44.4859619140625, -40.14354705810547, 25.19717788696289, -16.486225128173828, -36.4378776550293, -21.78931427001953, 7.715579986572266, -28.935176849365234, -1.5358459949493408, 56.47382736206055, -3.9602766036987305, -15.124466896057129, 29.8974609375, -37.68745040893555, -59.33717727661133, -9.963221549987793, -44.945762634277344, 10.705111503601074, 78.54307556152344, -5.00117301940918, 11.412040710449219, -42.0399284362793, -14.14365291595459, -8.59025764465332, 16.620033264160156, 7.680290222167969, -15.221046447753906, 44.818607330322266, -13.7872896194458, -32.30463790893555, -33.72929000854492, 40.68024444580078, 29.821128845214844, 8.592962265014648, 24.9073486328125, -14.419160842895508, 84.36691284179688, 32.30126190185547, 15.783370971679688, 47.189109802246094, 4.539237022399902, -1.5881683826446533, -9.410466194152832, 30.79022979736328, -47.0616569519043, 9.556438446044922, 31.69317054748535, -79.12722778320312, 31.590084075927734, 12.690710067749023, 18.046772003173828, 30.726818084716797, 4.761070251464844, -4.7894487380981445, -32.97984313964844, -15.445343017578125, 17.868459701538086, -34.56010055541992, -14.424026489257812, 23.43169403076172, 14.161076545715332, 17.84090805053711, 23.5467529296875, 30.791175842285156, 12.904265403747559, 5.001709938049316, 10.64771842956543, -5.829137802124023, -53.95772171020508, -7.11569881439209, -19.67186164855957, 37.3278923034668, 20.412551879882812, -4.04693603515625, 10.182339668273926, 69.63036346435547, -21.5572509765625, -14.065363883972168, 11.482325553894043, -54.958702087402344, -4.148273468017578, -50.247291564941406, -20.783048629760742, 11.55964183807373, 10.232921600341797, 0.4803731441497803, 32.290740966796875, -23.282934188842773, -13.714426040649414, -3.939864158630371, -52.18449783325195, 8.053101539611816, -19.09346580505371, -21.603288650512695, 23.815996170043945, 4.679548740386963, -32.7718505859375, -68.28398132324219, -51.841670989990234, 8.73832893371582, 21.309226989746094, -44.62409210205078, 18.77143669128418, -28.523591995239258, -28.848735809326172, -60.320735931396484, -24.49692153930664, -5.628810405731201, 21.09922981262207, -59.90636444091797, 4.731165409088135, -16.910053253173828, -46.139835357666016, -64.37188720703125, -7.229104518890381, -17.119897842407227, -30.479764938354492, -18.527301788330078, -41.58796310424805, 54.29825973510742, 24.869281768798828, -8.131163597106934, 23.04393196105957, 23.41482925415039, 55.81441116333008, 3.9822230339050293, -46.70220947265625, 29.255935668945312, 14.173653602600098, -15.378752708435059, -24.732885360717773, -27.056137084960938, -3.820099115371704, 21.177536010742188, 21.722991943359375, 23.280725479125977, 26.176549911499023, 22.928979873657227, -29.76995086669922, -19.057390213012695, 14.916653633117676, 12.52200698852539, -44.63301086425781, -76.898193359375, 4.106287002563477, 7.667449474334717, -4.9752020835876465, -6.4977874755859375, 20.58186912536621, 29.26066780090332, -29.380420684814453, -37.58457946777344, 4.541398048400879, 67.64264678955078, -1.9987390041351318, -18.194320678710938, 24.79353904724121, -21.062847137451172, -22.782039642333984, -66.90721130371094, -8.171978950500488, -27.893829345703125, 52.26188659667969, -17.360488891601562, -16.311086654663086, 51.07523727416992, 9.31828784942627, 26.432456970214844, -46.17306137084961, 25.827253341674805, -40.83300018310547, -26.901031494140625, -22.73115348815918, -3.7446043491363525, -12.011359214782715, 43.22468185424805, -42.68728256225586, -20.535341262817383, -37.980281829833984, 40.32557678222656, -36.315608978271484, 12.556097984313965, -4.452459812164307, 8.633572578430176, -29.235044479370117, -71.3590087890625, -42.044918060302734, 4.189671039581299, 63.676456451416016, -87.72795867919922, -17.7464599609375, -2.215811014175415, 13.167027473449707, -28.040300369262695, -36.68476867675781, 25.3692684173584, 40.1396484375, -13.661006927490234, -15.487625122070312, 40.381591796875, -64.3720474243164, 10.456975936889648, -21.79366683959961, -3.6902287006378174, 20.04486656188965, 12.654618263244629, 20.22626304626465, 4.10353946685791, -3.954423189163208, -42.600074768066406, -2.1867494583129883, -9.728522300720215, -31.61313819885254, -19.011123657226562, 25.910621643066406, -46.728240966796875, 9.49506950378418, -28.777990341186523, 7.655285835266113, -12.58041763305664, 36.57230758666992, -22.221590042114258, 24.655372619628906, 27.829822540283203, -31.58296775817871, 12.980134963989258, 17.454837799072266, 11.713878631591797, 7.650641918182373, 51.45292663574219, -19.510581970214844, -52.825557708740234, -7.691924571990967, -29.406347274780273, 7.485716819763184, -18.416236877441406, -43.70021438598633, 25.451404571533203, 28.953718185424805, 31.250282287597656, 16.567020416259766, -3.4237568378448486, 4.681763172149658, -26.1441707611084, 32.40119552612305, -13.342561721801758, 8.954036712646484, 10.778108596801758, -16.582775115966797, 7.3044538497924805, 27.56513023376465, 4.211442947387695, 17.38840675354004, 10.827091217041016, -40.41408920288086, -42.235538482666016, 39.16859817504883, -51.768585205078125, -14.943641662597656, -1.9238080978393555, 39.161312103271484, 24.06543731689453, 20.61153793334961, 42.45064926147461, -5.2521772384643555, -8.506843566894531, -12.51534652709961, -16.51495933532715, 22.183757781982422, -44.13977813720703, -17.232376098632812, 7.43026876449585, 51.27383804321289, 24.67816925048828, 68.51573181152344, 15.155529022216797, -22.743122100830078, 50.421043395996094, -77.76103973388672, 21.601533889770508, -7.62769889831543, -38.311973571777344, -11.96756362915039, -0.18799430131912231, -9.25844955444336, -46.169891357421875, 26.513452529907227, -15.582036018371582, -13.10353946685791, -0.31015700101852417, -86.35442352294922, 27.182544708251953, 26.44959259033203, 41.24507141113281, -16.718164443969727, 5.630023956298828, -13.918644905090332, -12.047853469848633, -14.380252838134766, -14.06446647644043, -10.013041496276855, -38.94559097290039, -35.03847122192383, -50.82476806640625, -29.307373046875, 0.8958394527435303, -63.8207893371582, -1.6520673036575317, 86.51764678955078, 23.518951416015625, 62.99102783203125, 10.775472640991211, 12.213833808898926, 26.06073760986328, -3.265291690826416, 24.512325286865234, -18.02792739868164, 24.222631454467773, 11.124879837036133, 19.948925018310547, -9.493476867675781, -45.6580696105957, -5.87747859954834, 10.895718574523926, -35.79821014404297, -43.78953170776367, 12.848062515258789, 31.13238525390625, 1.757116675376892, -38.154090881347656, -42.219879150390625, -19.049297332763672, -17.7484130859375, -2.5622711181640625, 16.15481948852539, -6.710150241851807, -9.86156940460205, -35.442501068115234, -31.80875015258789, 22.976360321044922, -28.40401840209961, -5.563140869140625, -12.71860122680664, 41.50926971435547, 9.554348945617676, -10.034750938415527, 10.237589836120605, -40.0438346862793, 27.547182083129883, 9.55645751953125, -71.41084289550781, -3.102315664291382, -38.18819046020508, 7.813442230224609, -14.018760681152344, -7.917620658874512, 47.93379592895508, -43.58164596557617, 37.491973876953125, -60.43183517456055, -38.09325408935547, 28.744895935058594, -28.742874145507812, 22.374406814575195, 2.892585039138794, 2.3715953826904297, 25.90989875793457, -1.142096757888794, -19.278579711914062, -29.31856346130371, -17.02012062072754, -30.92742919921875, -22.55146026611328, -22.135662078857422, -83.92630767822266, -23.78575897216797, 6.6824517250061035, 2.0990095138549805, -42.33369064331055, 53.89573287963867, -24.540990829467773, -38.99020767211914, -28.04712677001953, -40.47792053222656, 28.9388427734375, -0.587692379951477, -21.298608779907227, -18.17351722717285, 0.308231383562088, 8.241706848144531, 6.252314567565918, 52.862239837646484, 15.331629753112793, 23.95766258239746, -35.55522537231445, -12.073248863220215, -13.041793823242188, 38.48114013671875, 36.426265716552734, 19.463388442993164, -17.913785934448242, 81.73722076416016, 46.54315185546875, 37.836612701416016, 16.00609016418457, -28.449207305908203, 29.226539611816406, 9.236385345458984, 18.14217185974121, 25.778533935546875, -10.01456069946289, -9.29995346069336, 3.5154149532318115, 5.29457426071167, -15.064712524414062, 10.709809303283691, -64.18373107910156, 6.7432780265808105, 20.820653915405273, 15.220744132995605, -25.649227142333984, 19.732467651367188, -16.20760726928711, -2.0410454273223877, 21.776639938354492, 23.73126220703125, -27.33504295349121, -11.971272468566895, -21.80864715576172, -5.107639312744141, 19.892122268676758, 8.480578422546387, -32.05111312866211, -32.33877182006836, -45.206024169921875, -36.953426361083984, 43.494815826416016, -30.673059463500977, 29.368648529052734, -36.978389739990234, -0.3149455189704895, -2.7274155616760254, 1.6201705932617188, 38.55809783935547, -50.85060119628906, -25.720897674560547, 11.197256088256836, -28.452484130859375, -31.30535316467285, -22.143587112426758, -84.63374328613281, -48.3306770324707, -9.557947158813477, -29.47538185119629, 16.58989143371582, 0.048307374119758606, -14.916007995605469, 9.08005142211914, -3.9743552207946777, -32.34761428833008, -15.640928268432617, -28.236995697021484, -23.48200225830078, -46.406654357910156, -15.667574882507324, 20.29971694946289, -34.81960678100586, 4.26521110534668, 14.814441680908203, 13.881532669067383, -58.13747024536133, -24.237218856811523, -17.145811080932617, -8.650270462036133, -23.270456314086914, 35.5231819152832, 10.569657325744629, 6.8307390213012695, 24.276811599731445, -5.075311183929443, -44.504974365234375, -6.032196998596191, 11.592853546142578, 10.331457138061523, -12.07639217376709, -31.84020233154297, -1.7041305303573608, 66.03025817871094, 4.946455001831055, 13.567643165588379, -59.05984115600586, 11.760052680969238, -50.178253173828125, 70.19125366210938, 2.8727505207061768, -87.38993835449219, 63.98210144042969, -25.805198669433594, 37.41038513183594, 13.390321731567383, -26.991235733032227, -25.19784164428711, 17.41075897216797, 20.49547004699707, 1.2775852680206299, -34.234527587890625, 18.82059669494629, 55.38520812988281, 17.013078689575195, 17.69894790649414, 25.93975257873535, 14.801397323608398, 21.710691452026367, 23.047462463378906, -21.729267120361328, 11.077038764953613, -34.6935920715332, -0.9452818632125854, 26.94342803955078, 8.917600631713867, -12.280470848083496, 42.52662658691406, -17.699493408203125, -25.04230499267578, 4.415923595428467, -97.05622100830078, 14.700560569763184, 11.980137825012207, 91.11396026611328, -9.491780281066895, -3.7099950313568115, 32.18587112426758, 31.868040084838867, 12.913581848144531, -27.301767349243164, 33.601402282714844, 7.900493144989014, -72.6593246459961, 8.982566833496094, -42.110870361328125, 5.165043830871582, -4.283064842224121, 7.683443546295166, -7.923971176147461, -77.38484191894531, -7.776638507843018, 57.175331115722656, 14.789780616760254, 25.564828872680664, 70.21358489990234, 0.46362724900245667, 2.357042074203491, 1.8072339296340942, -24.50452423095703, -27.331819534301758, 53.063045501708984, 30.6605281829834, -28.77518653869629, 19.03386116027832, -4.426781177520752, 5.861481666564941, -33.262474060058594, 44.365684509277344, 11.069490432739258, -43.364036560058594, 32.256717681884766, -35.24873352050781, 27.066478729248047, 13.851037979125977, 3.428675413131714, -16.131393432617188, -17.897430419921875, -32.38405990600586, -66.04116821289062, 14.268683433532715, -26.32056999206543, -17.082679748535156, 51.44410705566406, 37.55622863769531, 53.80213165283203, -19.857433319091797, -13.897285461425781, 7.576314926147461, 20.501365661621094, 26.586145401000977, 34.30332565307617, 8.82023811340332, 7.616352558135986, -26.886980056762695, -34.58831024169922, 16.14130210876465, -15.547162055969238, 42.64488983154297, -23.105255126953125, -23.71467399597168, 61.71305847167969, -22.07408905029297, 1.1062523126602173, 13.225398063659668, 30.58933448791504, 11.302766799926758, 3.55155348777771, -52.14219284057617, 11.918264389038086, -24.300477981567383, -21.978870391845703, -14.905847549438477, -49.145416259765625, -63.777320861816406, 14.93876838684082, -16.108827590942383, -26.018360137939453, 17.03816795349121, -24.796100616455078, 28.376577377319336, 14.838436126708984, -8.806741714477539, 73.19588470458984, -5.148375988006592, 12.129012107849121, -20.983720779418945, -6.846960067749023, 31.217500686645508, 60.380714416503906, 38.32744598388672, 8.993728637695312, -2.236936330795288, -12.724482536315918, 19.76002311706543, 19.376388549804688, -18.779281616210938, -12.651787757873535, -25.914560317993164, 18.903533935546875, -4.624882698059082, -19.422672271728516, -7.237117767333984, 4.990478992462158, 23.058515548706055, -17.75920867919922, 21.708772659301758, -49.49455642700195, -12.8482084274292, -37.912132263183594, -22.1873836517334, -18.35923957824707, 10.874336242675781, -41.06555938720703, -20.871477127075195, 13.917341232299805, 7.204960346221924, -9.195924758911133, 41.13569259643555, -20.499500274658203, -32.24323272705078, -50.878639221191406, 3.3283064365386963, 33.56687545776367, -11.294052124023438, 25.35958480834961, -29.94873809814453, -14.66841983795166, -13.314300537109375, 46.437496185302734, -36.25773239135742, -16.582515716552734, -7.921016693115234, 1.1150949001312256, -53.14299774169922, 4.191286087036133, 24.216081619262695, -7.603647708892822, -12.817142486572266, 17.079191207885742, -49.65388870239258, -17.38490104675293, 24.12758445739746, -8.656217575073242, -25.466672897338867, 47.13563537597656, 8.347966194152832, -2.6533915996551514, -7.884958744049072, -0.17560482025146484, -5.940084934234619, -3.1638922691345215, 1.1681687831878662, -33.30780792236328, 37.91549301147461, -24.9577693939209, 3.1082093715667725, -7.142722129821777, 85.3701171875, 14.664793014526367, 78.02073669433594, -9.494078636169434, -6.923808574676514, 2.3587656021118164, -6.207986354827881, -36.72932052612305, 28.984798431396484, 25.58136749267578, -46.35818099975586, 36.54226303100586, -30.185720443725586, 61.134395599365234, 41.04197311401367, 28.26598358154297, 3.505953550338745, -13.375017166137695, 3.2084567546844482, -20.78862762451172, -39.88984680175781, 22.634906768798828, 23.032649993896484, 2.837428569793701, -9.920687675476074, 16.804277420043945, 23.33895492553711, -22.14645004272461, -8.651507377624512, 50.93326187133789, 19.049171447753906, 9.366043090820312, 18.23866081237793, 45.228633880615234, -30.567873001098633, -10.234835624694824, 19.796911239624023, 11.89976978302002, -7.850069522857666, 14.329205513000488, 10.100950241088867, -31.987512588500977, 9.003584861755371, 7.1485090255737305, 45.48542785644531, 48.258949279785156, -13.469053268432617, -87.19635009765625, 21.555322647094727, -3.804718494415283, -39.879302978515625, 25.360563278198242, 33.62449645996094, -42.7638053894043, 23.922595977783203, -39.22981262207031, -8.1034574508667, 18.13856315612793, -36.75109100341797, -52.65534591674805, 1.282536506652832, 19.330284118652344, -22.951175689697266, 11.44825553894043, -28.09864044189453, -14.47513198852539, -20.56696891784668, 21.230449676513672, -23.63373565673828, 68.89277648925781, -49.761085510253906, -18.962825775146484, 72.42391967773438, 5.172067642211914, -17.57672882080078, -37.39202880859375, 33.861106872558594, 9.780102729797363, -35.07353210449219, -6.327704429626465, -11.474554061889648, -61.21152877807617, -4.668055534362793, -8.676314353942871, -32.8334846496582, -14.69076156616211, 14.115578651428223, 7.962989330291748, -3.815260887145996, -29.675573348999023, 1.9543274641036987, 43.902915954589844, -41.58399963378906, -8.133387565612793, 36.84119415283203, -0.23934940993785858, -31.449909210205078, -49.66431427001953, 26.648056030273438, 0.1837899088859558, 17.9211483001709, 9.055291175842285, 3.7538645267486572, 5.298142910003662, -30.904544830322266, 28.62075424194336, -30.277790069580078, 22.03189468383789, -10.677116394042969, -7.545104026794434, -18.77777099609375, -2.006359100341797, -12.203852653503418, -43.774600982666016, -3.1559336185455322, 4.427514553070068, -11.317008018493652, 41.70978546142578, 17.264293670654297, 0.16864757239818573, -1.6605907678604126, 4.64694356918335, 11.785931587219238, 23.132326126098633, 26.614404678344727, 41.08011245727539, 2.6017708778381348, 14.37028980255127, 35.41462707519531, 6.9593353271484375, 38.690216064453125, 15.366495132446289, -35.29033660888672, -8.968771934509277, 20.68597412109375, 60.24547576904297, 9.73662281036377, -17.665203094482422, -28.24774169921875, -33.099693298339844, -18.688335418701172, 14.635475158691406, 28.93742561340332, -3.2549803256988525, 80.40461730957031, -17.384334564208984, 33.12406539916992, 41.47922134399414, -14.602335929870605, -13.139917373657227, -7.2442240715026855, 12.958023071289062, -24.266021728515625, 65.55476379394531, 6.889312744140625, -25.196592330932617, -18.131444931030273, -31.913297653198242, 9.550469398498535, 24.628158569335938, 10.494135856628418, 61.15993118286133, 4.448214054107666, 27.02935791015625, -40.5181770324707, -26.703405380249023, -20.314870834350586, 9.700486183166504, -21.973852157592773, -2.9300808906555176, 45.54419708251953, -10.509111404418945, -18.572757720947266, 22.21352195739746, 4.769190311431885, 39.97230529785156, -42.01874542236328, 3.3883349895477295, -2.1732709407806396, -62.568031311035156, -22.794158935546875, -11.206876754760742, 19.079668045043945, -21.962602615356445, -6.367185115814209, -1.5566465854644775, 58.94225311279297, -25.749156951904297, 22.441492080688477, -14.924078941345215, 5.525744438171387, 16.24007797241211, -37.235538482666016, -0.06811457127332687, -41.41826629638672, -9.458773612976074, -75.5130386352539, -25.333229064941406, -35.483123779296875, 10.124288558959961, 12.440080642700195, 4.138620376586914, 3.046684741973877, 15.145646095275879, -14.751598358154297, -33.654972076416016, -22.994476318359375, 3.5573294162750244, 16.195724487304688, -22.786540985107422, 28.377124786376953, -14.76922607421875, 21.563629150390625, -15.923638343811035, -13.303220748901367, -2.4946138858795166, 5.056858539581299, 30.361278533935547, 32.46962356567383, 1.6974698305130005, -20.68730926513672, -20.40203094482422, -20.979578018188477, 18.498790740966797, -16.22830581665039, 10.951932907104492, 3.779383659362793, -2.959498882293701, -34.45029067993164, -15.65247631072998, 5.277998447418213, 7.127265930175781, -40.40110778808594, -13.137123107910156, -10.487987518310547, 41.570064544677734, 18.00528335571289, 17.4256649017334, -9.370945930480957, 5.081196308135986, -53.36760330200195, 37.718040466308594, -89.17062377929688, -2.8959567546844482, 39.59492111206055, 28.173072814941406, -66.5183334350586, -27.82200050354004, 11.18447494506836, -38.40835952758789, 24.722312927246094, -17.473350524902344, 43.931663513183594, 31.303600311279297, -61.655372619628906, 14.357879638671875, -25.17481231689453, 26.933021545410156, 17.555479049682617, 12.959468841552734, 20.821651458740234, -7.310247421264648, 32.867027282714844, -20.282005310058594, -11.544914245605469, -16.18697738647461, -42.041160583496094, -8.863814353942871, 9.834651947021484, -33.44003677368164, 19.247005462646484, -32.76332092285156, -19.198518753051758, 58.92277908325195, -20.660490036010742, 20.16067123413086, -16.79827880859375, -37.198673248291016, -35.04301834106445, -42.14323806762695, -84.94047546386719, 3.2830188274383545, 63.518680572509766, 20.630491256713867, -15.68558120727539, 31.713768005371094, 60.45287322998047, -13.756534576416016, -13.551241874694824, -1.657596230506897, -51.27482223510742, 14.818075180053711, 56.975379943847656, -18.187225341796875, -33.733619689941406, -5.599069118499756, 40.3718147277832, 42.05916976928711, -15.461996078491211, -43.44709396362305, -46.26007080078125, 10.725509643554688, 51.30818176269531, -43.49592590332031, 13.789981842041016, 44.06280517578125, 10.935914993286133, -26.547988891601562, 7.098544597625732, 16.258277893066406, -10.542969703674316, 66.96107482910156, 32.81010818481445, -7.389895439147949, -22.727537155151367, 6.079174995422363, -62.12479019165039, -13.161412239074707, 63.19719314575195, 66.17703247070312, 21.53548240661621, -24.59261703491211, 38.1778564453125, -31.019729614257812, -5.442266941070557, -51.497711181640625, 36.663597106933594, -1.0132203102111816, -2.3535683155059814, -1.6867879629135132, 2.3635761737823486, 42.37625503540039, 7.558712482452393, -47.88197708129883, -4.345375061035156, 7.7589263916015625, -0.8688914179801941, -28.906930923461914, 8.939142227172852, -20.135032653808594, -51.070045471191406, -11.023957252502441, 43.35590744018555, 11.597431182861328, 9.907864570617676, 37.26710510253906, 35.644954681396484, -1.7208517789840698, -2.719304084777832, -7.363105297088623, 6.164280414581299, 30.308467864990234, -55.33090591430664, -63.197120666503906, -0.07685016095638275, -16.901844024658203, 7.452980041503906, 25.314165115356445, 22.46416473388672, 8.543621063232422, 11.63814640045166, 28.454853057861328, 23.652523040771484, -15.493796348571777, 35.22227096557617, -25.03731918334961, -6.196731090545654, 13.01539134979248, -8.710372924804688, -14.64289379119873, 5.101588249206543, -42.44804763793945, 27.14006996154785, -23.458263397216797, 5.3214569091796875, -5.212626934051514, 6.0052103996276855, 49.13391876220703, 38.840476989746094, -9.628249168395996, 15.45885944366455, -35.70853042602539, -37.07265853881836, 37.81393814086914, -31.413148880004883, -31.693683624267578, 56.29386520385742, 38.82809066772461, 20.095169067382812, -67.96443939208984, -39.93732833862305, -10.388788223266602, -12.109694480895996, 2.431929111480713, 50.50421905517578, -67.96219635009766, 3.8718862533569336, -11.812870979309082, 28.81064224243164, -40.09547805786133, 24.0936222076416, 1.6959396600723267, 39.82185745239258, 23.779481887817383, 27.23739242553711, -51.19694137573242, -17.357913970947266, 36.373111724853516, 1.2002135515213013, 49.974788665771484, -25.282203674316406, 28.711233139038086, 59.40264892578125, -2.3297414779663086, -42.51780700683594, -2.7937893867492676, -12.575922012329102, 47.5153923034668, -2.8309173583984375, 15.871567726135254, 74.87039184570312, -25.083452224731445, -31.029470443725586, -16.295265197753906, -35.61781692504883, -31.940542221069336, 12.391580581665039, -22.992401123046875, -85.3886489868164, -28.11956787109375, 6.091739177703857, -3.5792620182037354, -26.26075553894043, 32.213340759277344, -5.355317115783691, -3.3469481468200684, -11.690709114074707, 54.27579879760742, -60.78042984008789, -0.6334381103515625, -21.32316017150879, 21.370956420898438, 47.26080322265625, -32.405792236328125, 11.585550308227539, 21.268457412719727, -12.77790641784668, 16.494108200073242, 3.20723557472229, 7.1750640869140625, 37.25305938720703, -41.38787078857422, 30.418203353881836, 26.753774642944336, -7.464937210083008, -16.900468826293945, -52.24824523925781, -15.987403869628906, 28.145217895507812, -21.79207420349121, -38.210086822509766, 68.11174011230469, 8.974635124206543, 23.898296356201172, 3.689517021179199, -26.118078231811523, 9.931228637695312, -8.596712112426758, 2.9074976444244385, 23.204845428466797, -26.313613891601562, 30.51500129699707, -27.023096084594727, -61.23237991333008, 13.274357795715332, 5.276773452758789, 12.472624778747559, -6.569357872009277, -7.620532035827637, -28.030317306518555, -16.622812271118164, -14.423030853271484, 53.648216247558594, 18.2025146484375, 23.800865173339844, 45.36740493774414, -16.370887756347656, -33.803932189941406, 35.580997467041016, 10.39797592163086, -27.80812644958496, 6.140082359313965, -24.825946807861328, 36.04795837402344, 28.842985153198242, 5.87437105178833, 40.474552154541016, -11.112399101257324, 57.34955978393555, -64.84297943115234, 17.235200881958008, -2.680304765701294, 8.070241928100586, -16.78795623779297, -17.963632583618164, 11.762932777404785, 3.3533453941345215, 57.31395721435547, 42.795963287353516, 28.678585052490234, 0.6546238660812378, 12.29383373260498, 6.002762317657471, 7.138372421264648, -26.46802520751953, 15.866430282592773, -37.19919204711914, -35.383941650390625, 48.57378387451172, 14.342811584472656, 11.008465766906738, -19.221328735351562, 5.835115909576416, -3.425980567932129, -22.132780075073242, 28.347667694091797, -35.791969299316406, 32.098690032958984, 26.628347396850586, 79.26468658447266, 36.35304260253906, -3.6500325202941895, -66.57695770263672, -24.169126510620117, -0.20290079712867737, 26.180892944335938, 23.189868927001953, 44.7818717956543, 28.248458862304688, -6.5941877365112305, -38.357200622558594, -28.571292877197266, -10.293254852294922, -9.23936653137207, 3.684605598449707, 9.783515930175781, 47.94483947753906, -6.247103214263916, 52.086544036865234, 26.728742599487305, 12.519963264465332, 19.94352912902832, 4.073720932006836, -32.861759185791016, -4.70485258102417, -10.488298416137695, -16.882627487182617, 14.285252571105957, 27.686716079711914, 30.3200740814209, 32.17903518676758, 0.7670862674713135, -25.385751724243164, 64.63143157958984, 19.859294891357422, 21.16880226135254, 18.515840530395508, -14.683411598205566, 8.52753734588623, 40.80229187011719, -12.962449073791504, -38.7574577331543, 4.436734199523926, -2.2333319187164307, 20.08011245727539, -41.44593048095703, -17.43147850036621, -22.83727264404297, -44.28390884399414, -26.71828842163086, -87.62832641601562, -27.031482696533203, -0.6305388808250427, -11.386857032775879, 34.03976821899414, -21.54163360595703, -19.558853149414062, 40.93540954589844, -39.48036575317383, 29.941619873046875, 17.89306640625, 33.266136169433594, 14.48995304107666, -47.93537902832031, -35.386226654052734, 40.9513053894043, -7.826777935028076, 2.5548696517944336, 15.420877456665039, 30.532075881958008, 21.480430603027344, -3.9290928840637207, 11.82284927368164, 10.950862884521484, 40.403907775878906, -73.48906707763672, -17.001583099365234, 44.04073715209961, -4.371515274047852, 2.6138100624084473, -28.534107208251953, -42.23029708862305, -31.539762496948242, -32.45747756958008, -8.764456748962402, -31.6722354888916, -40.52866744995117, -25.21963882446289, 11.349729537963867, 16.1175479888916, 12.886778831481934, 28.257875442504883, -36.1248779296875, 1.3712184429168701, -48.46778869628906, 31.819320678710938, -6.8886637687683105, 13.687459945678711, 32.10355758666992, -16.212574005126953, -13.826444625854492, -34.09605026245117, 5.730215072631836, 1.270512342453003, 19.994977951049805, 1.8099339008331299, 7.126718521118164, -11.292693138122559, 24.575429916381836, -19.974027633666992, -44.81946563720703, -64.86631774902344, -16.4860782623291, 12.465511322021484, 39.77619552612305, -32.749847412109375, 27.18108558654785, -8.006614685058594, 21.80961799621582, 30.90450096130371, 63.62733459472656, 9.06152629852295, 8.356953620910645, -7.115232944488525, -22.722013473510742, 7.625139236450195, 28.694808959960938, 26.727176666259766, -9.45311450958252, -10.107566833496094, -31.328123092651367, 12.199448585510254, -31.30389976501465, -27.258182525634766, 33.745330810546875, 10.765819549560547, -14.908843040466309, 1.7535443305969238, 62.637977600097656, 23.999961853027344, 62.463775634765625, -5.206029891967773, -7.478023052215576, -58.70125961303711, -9.080619812011719, 40.72641372680664, 70.3576431274414, 5.475381374359131, 8.576335906982422, -33.83214569091797, 17.860105514526367, 49.03408432006836, 39.852603912353516, -13.864655494689941, -46.570556640625, 13.129903793334961, -41.0861930847168, -27.94039535522461, 20.144607543945312, -33.75080108642578, -15.456478118896484, 2.3782155513763428, -12.23332405090332, -7.158652305603027, 25.420196533203125, -5.018307209014893, 27.410938262939453, 37.54654312133789, 6.557786464691162, -11.738319396972656, -21.938785552978516, -20.05143928527832, -3.4663760662078857, 32.6937141418457, -17.1805362701416, -62.1560173034668, 6.495509624481201, 1.301545262336731, -7.962581634521484, 15.668269157409668, 25.687902450561523, -28.55451202392578, 6.933675289154053, -1.483394980430603, 12.840502738952637, -1.7393509149551392, 0.2976820468902588, -68.87171936035156, 42.69404602050781, 29.12567710876465, -6.779015064239502, 3.645310640335083, -92.46052551269531, -1.749416708946228, -34.238807678222656, 20.83251190185547, -27.487897872924805, 25.957901000976562, 7.769650459289551, -20.96514129638672, 18.297489166259766, -21.579389572143555, 22.744020462036133, -28.083141326904297, 17.586946487426758, -5.049408912658691, 25.091087341308594, 42.25480270385742, -62.65538024902344, 1.000144362449646, -28.97983169555664, 46.32843017578125, -61.22118377685547, 18.062501907348633, -85.19563293457031, 26.045330047607422, 9.715566635131836, -4.341555595397949, 51.70254898071289, -3.029890537261963, 17.62814712524414, -36.958309173583984, 10.08205795288086, 27.839136123657227, 69.4898681640625, -25.06464958190918, -0.41838645935058594, 53.029232025146484, 76.77621459960938, 9.63466739654541, 16.257007598876953, -45.582984924316406, 11.753619194030762, -55.46195983886719, -25.219572067260742, 18.03441619873047, -24.817550659179688, -44.08784866333008, 84.24703216552734, -58.017024993896484, 29.366907119750977, 25.552112579345703, -17.76182746887207, 20.529369354248047, -12.388792991638184, -35.13882827758789, -13.3632173538208, -8.812664031982422, 31.251251220703125, 19.910938262939453, -25.599740982055664, 45.61822509765625, -48.063560485839844, -60.284393310546875, -44.84568405151367, -24.373933792114258, -15.810699462890625, -34.67626953125, -4.103845119476318, 2.7747018337249756, 42.84416198730469, -23.383743286132812, -1.5887304544448853, -25.426877975463867, 21.037382125854492, -8.245336532592773, 72.94606018066406, -11.289030075073242, 47.877017974853516, -4.3202385902404785, -25.448312759399414, 22.135438919067383, 56.03696060180664, -11.45065975189209, 30.021270751953125, 32.89677810668945, 0.15009933710098267, -5.561704158782959, 26.956787109375, 48.79821014404297, 25.626264572143555, 13.537550926208496, -54.83933639526367, -29.420019149780273, -21.840639114379883, 11.912904739379883, 21.50653076171875, -34.81876754760742, 11.252182960510254, -4.873739719390869, -35.67267990112305, -10.637913703918457, -13.915061950683594, -1.9238131046295166, 57.023475646972656, -0.21498194336891174, -14.6956205368042, 30.66681480407715, 11.607897758483887, 14.848737716674805, 21.303754806518555, 22.257972717285156, 81.52940368652344, -26.233505249023438, 10.984147071838379, 5.860232353210449, -31.447261810302734, -9.5835599899292, 47.08527755737305, 39.94857406616211, -10.461240768432617, 32.32791519165039, -5.713626384735107, 3.2054214477539062, -42.776893615722656, 15.885563850402832, -14.7177095413208, -27.35887908935547, 5.771392822265625, -24.552946090698242, 8.126392364501953, -34.0021858215332, -62.42689895629883, -25.550172805786133, 13.290783882141113, 58.708518981933594, -34.73983383178711, -5.607831954956055, -44.90045928955078, 7.990973949432373, 10.662308692932129, -12.30103874206543, 23.968191146850586, 14.276921272277832, -0.0937613695859909, -39.312339782714844, 19.184139251708984, 0.7665897607803345, -38.04020309448242, 1.066890835762024, -24.347862243652344, -26.74264907836914, -47.465248107910156, 3.6182355880737305, 20.928272247314453, 0.2274763286113739, -12.399405479431152, 33.579017639160156, 5.084533214569092, -35.75742721557617, -41.302486419677734, -65.90185546875, -58.54934310913086, -12.594603538513184, -39.297115325927734, -5.657972812652588, 8.421974182128906, 36.37007522583008, -25.618942260742188, -0.27617543935775757, 43.97829055786133, -9.98775577545166, 0.5190954208374023, 10.049735069274902, -26.078855514526367, 12.835749626159668, -31.626279830932617, -30.096702575683594, -26.610809326171875, -13.283187866210938, 15.5696439743042, -31.08529281616211, -31.635814666748047, 24.673601150512695, 15.352371215820312, -0.1415749192237854, 45.69495391845703, 10.240689277648926, 34.821937561035156, 27.6094913482666, 4.042811870574951, -22.463062286376953, 3.207721710205078, -27.067167282104492, 31.164228439331055, 25.737388610839844, -5.099499225616455, -68.04208374023438, 0.1612040102481842, 37.983558654785156, -53.909175872802734, 13.987725257873535, 29.89165496826172, 12.06019115447998, 0.08756756782531738, -46.086585998535156, 64.44149780273438, -14.00144100189209, -44.22319793701172, 15.711150169372559, -10.898805618286133, 33.8031005859375, 3.9103589057922363, -12.796331405639648, -8.948609352111816, -20.647361755371094, 22.559650421142578, 11.631003379821777, 29.37872886657715, -53.163414001464844, 34.38164138793945, 0.9322453141212463, -33.2122802734375, 8.268808364868164, -43.493003845214844, 19.663694381713867, -20.27865982055664, 20.278127670288086, 8.943583488464355, -41.39624786376953, -35.606449127197266, 15.819284439086914, 25.977373123168945, -39.146087646484375, -7.997786521911621, 0.4917546808719635, -40.399173736572266, 24.4858455657959, -41.73291015625, -0.8802196383476257, -44.688011169433594, -47.37234878540039, -35.5379638671875, 33.428131103515625, -42.834259033203125, -23.421586990356445, 19.636648178100586, -7.8739213943481445, 21.907634735107422, -33.766353607177734, 17.918136596679688, -19.948020935058594, 13.904317855834961, -54.57939910888672, -41.29022979736328, -49.38654708862305, 21.120800018310547, 15.790699005126953, 10.903274536132812, -5.867837429046631, 30.731578826904297, -9.055598258972168, -13.143986701965332, 16.192829132080078, 16.244613647460938, -27.489633560180664, -15.68259048461914, 8.877973556518555, 12.308245658874512, -4.761486530303955, 34.13725280761719, -55.726165771484375, -11.557754516601562, 22.84090232849121, -31.257923126220703, 33.74555969238281, -2.3484957218170166, 48.16147232055664, 19.8614559173584, 25.005176544189453, -3.8483901023864746, 37.4962272644043, -19.793264389038086, -2.508185386657715, 5.965182304382324, -7.627254486083984, 28.457199096679688, 26.611825942993164, -28.758197784423828, -18.343416213989258, -28.4930362701416, 2.110229253768921, 7.301582336425781, 33.16223907470703, -15.685035705566406, 28.6136474609375, 25.10702896118164, 34.02840042114258, 5.727077960968018, 2.611510753631592, -15.890813827514648, -4.682738780975342, 8.689942359924316, -5.186661243438721, 5.667393684387207, 54.9769287109375, 34.494197845458984, -13.253236770629883, -2.7279412746429443, -20.422969818115234, -8.632547378540039, -1.032297968864441, 12.912851333618164, 23.69426155090332, 8.29298210144043, 22.690746307373047, -93.18882751464844, -51.48194885253906, -50.779537200927734, 29.55164337158203, -64.70156860351562, -1.7511332035064697, -0.6383375525474548, 6.480819225311279, 5.9977288246154785, -9.759709358215332, -20.102779388427734, -69.62030029296875, 89.14740753173828, 6.0215630531311035, 7.027527332305908, 29.988805770874023, -10.863321304321289, -50.64601135253906, -23.076326370239258, -36.83098602294922, -1.4880743026733398, 29.784343719482422, 15.102034568786621, 24.20166015625, 58.995758056640625, 13.019224166870117, 8.12282943725586, -34.32606506347656, -19.8026123046875, 46.497459411621094, 29.167930603027344, -11.881271362304688, -8.048360824584961, -34.44682693481445, -32.59571075439453, 16.276260375976562, 3.520966053009033, -32.12593078613281, -10.22713851928711, -33.109195709228516, 16.481739044189453, -30.334882736206055, 29.079072952270508, 25.530668258666992, -3.6385409832000732, -15.88746452331543, 10.774564743041992, -2.078690528869629, 25.977693557739258, -10.933838844299316, -14.5606107711792, -24.985612869262695, -27.244783401489258, 6.528005123138428, -45.421226501464844, -40.205230712890625, 24.6297550201416, 15.213616371154785, 47.73134231567383, -4.07651948928833, -40.72054672241211, 59.616920471191406, -6.995731830596924, 21.673351287841797, 19.131895065307617], [-12.880645751953125, -37.731441497802734, -2.0843050479888916, 48.444091796875, 15.552631378173828, 19.81247329711914, -4.925826549530029, -9.922626495361328, -22.728530883789062, -21.11868667602539, -28.125017166137695, 5.677392959594727, -71.06378936767578, -50.2065544128418, 43.92108154296875, -53.15612030029297, 13.490336418151855, -13.554945945739746, 18.712602615356445, -11.68127155303955, -12.126482009887695, 32.477996826171875, 22.600337982177734, -37.352237701416016, -40.35874557495117, -37.69221115112305, -21.00105857849121, 21.606128692626953, -18.593799591064453, 9.523089408874512, -13.106444358825684, 16.638994216918945, -9.970693588256836, -18.388553619384766, -7.453060626983643, 0.8978484869003296, 1.036321759223938, -3.283090114593506, 15.65227222442627, -19.985519409179688, -10.074628829956055, 16.8969783782959, 38.87998962402344, -1.4522292613983154, -6.084197044372559, -45.93060302734375, 69.4708251953125, 11.427371978759766, -28.032833099365234, 60.797454833984375, -12.809182167053223, -7.136875629425049, 8.398696899414062, -7.452512264251709, 3.969757080078125, -85.97997283935547, 2.8538832664489746, 12.879405975341797, 4.26597785949707, 9.188491821289062, 3.3124191761016846, 25.571617126464844, 0.4933279752731323, 41.50810241699219, 30.974925994873047, -25.285459518432617, 49.23536682128906, 43.17407989501953, -59.197723388671875, 69.73478698730469, -19.09384536743164, 25.94040298461914, -51.6414794921875, 23.8165283203125, 15.27695083618164, 37.55748748779297, 7.902566432952881, 10.525243759155273, -39.738468170166016, 6.640083312988281, -24.146198272705078, 4.911571025848389, -20.08216094970703, -29.875391006469727, 5.958041667938232, -41.827667236328125, 3.1874709129333496, -11.364264488220215, -32.490108489990234, -53.09953308105469, -1.6261327266693115, 38.58539962768555, -9.979066848754883, -46.954952239990234, -36.33778381347656, -1.2760534286499023, -7.841919898986816, -13.879366874694824, 49.62819290161133, 40.604583740234375, 2.595017671585083, 44.92384338378906, -41.42929458618164, 26.331647872924805, 0.43714189529418945, -45.318172454833984, -8.823583602905273, -2.004605293273926, -28.51155662536621, -44.246952056884766, 59.148475646972656, 6.135146141052246, 32.446258544921875, -7.909467697143555, 15.544172286987305, 56.537261962890625, -17.58357810974121, -45.09980392456055, -12.654610633850098, -25.0653133392334, 27.87582778930664, -60.097869873046875, 14.169540405273438, 35.021827697753906, 17.927427291870117, 7.603301048278809, -38.137550354003906, -7.110835075378418, 11.957179069519043, 18.37563705444336, -0.3315659165382385, -41.32398223876953, -15.034745216369629, 10.476486206054688, 20.045269012451172, 34.45952224731445, 64.4963150024414, 15.535558700561523, 7.977289199829102, -31.81998634338379, -23.345985412597656, 67.38536071777344, -12.87686824798584, -37.06379318237305, -31.235675811767578, -1.5500982999801636, 26.42137336730957, 13.754155158996582, -6.2338738441467285, -2.466015577316284, -22.304323196411133, -31.719745635986328, 21.124921798706055, -14.593770980834961, 22.376893997192383, 24.66403579711914, -11.69460391998291, 1.5934127569198608, 18.102842330932617, 41.583744049072266, 0.8308510184288025, -3.6479647159576416, 20.643190383911133, 21.453527450561523, -4.931755065917969, -13.865890502929688, 20.869577407836914, 17.834962844848633, -22.912885665893555, -55.110504150390625, 23.938112258911133, 7.719072341918945, 16.748334884643555, -55.26217269897461, -32.429168701171875, 31.811767578125, -6.686255931854248, -25.890073776245117, 14.501222610473633, 3.153186321258545, 50.35763168334961, 18.44649314880371, 11.718554496765137, -40.43242645263672, 14.980491638183594, 12.765275001525879, 1.9854793548583984, -46.510009765625, 4.279540061950684, 25.450489044189453, 23.267412185668945, 25.92856788635254, 49.28903579711914, -42.76579666137695, 6.773758888244629, 5.102622985839844, 27.028459548950195, -1.0838031768798828, 18.983116149902344, -40.592323303222656, 9.412375450134277, -34.26130294799805, 2.4931275844573975, 2.174156904220581, -29.4259090423584, 30.40715217590332, -38.41471481323242, -58.749847412109375, 15.618356704711914, -22.284440994262695, -12.320167541503906, 3.443223237991333, 11.949604988098145, 3.028498649597168, -42.24812316894531, 33.64440155029297, 36.761966705322266, -22.18779945373535, 15.759955406188965, 68.43600463867188, -5.759957790374756, 3.646879196166992, 25.687610626220703, 42.27543640136719, 25.203052520751953, 17.876672744750977, -34.87139892578125, -0.347601979970932, 5.468769073486328, -15.394329071044922, -18.713899612426758, -1.7731287479400635, -4.045135498046875, -11.478533744812012, 9.232681274414062, 20.449399948120117, 27.615333557128906, 15.171342849731445, 29.82485580444336, -29.63675880432129, 44.36027526855469, 9.339229583740234, 38.782413482666016, 36.3355598449707, 39.46687698364258, -28.06325340270996, -25.592674255371094, -49.971771240234375, 36.37868118286133, -6.529702186584473, -20.279346466064453, -19.413522720336914, -0.6456424593925476, 3.8453972339630127, 41.918701171875, -35.12192916870117, -31.821821212768555, 5.478701591491699, -33.2759895324707, -2.3432154655456543, 17.74551010131836, 2.3219025135040283, 26.77167320251465, 6.523519992828369, 6.884533405303955, 19.502656936645508, 12.393233299255371, -0.701666533946991, 41.22233581542969, -45.4591064453125, -9.011458396911621, -24.779315948486328, -60.935791015625, -74.48514556884766, -27.469764709472656, 17.62631607055664, 45.692169189453125, 18.797225952148438, -36.65666580200195, -50.24595260620117, 4.0425238609313965, -18.3217716217041, 39.02486801147461, -3.918602228164673, -31.611425399780273, 52.234039306640625, 17.43290901184082, -41.27336502075195, 15.354269981384277, 15.168328285217285, 30.179325103759766, -12.303136825561523, -0.3622767925262451, 22.168222427368164, -17.718034744262695, -24.678380966186523, -6.03959321975708, 47.19887161254883, 38.01970672607422, 10.035445213317871, -10.804190635681152, -15.462562561035156, 2.633117198944092, -8.310120582580566, 74.31077575683594, 14.661069869995117, -11.87619400024414, 28.7546443939209, 15.46138858795166, -5.209412574768066, -1.0358259677886963, -33.940513610839844, 18.652141571044922, 14.718141555786133, 4.678487300872803, -53.36830139160156, 12.707547187805176, -57.683712005615234, -21.181541442871094, -22.11511993408203, 4.125894546508789, -52.97142028808594, 15.5736083984375, 27.35211753845215, 19.45061683654785, -26.112319946289062, 4.4079742431640625, -61.477088928222656, -56.19971466064453, 67.85578918457031, -34.54853820800781, -6.3039960861206055, -15.824557304382324, 24.246599197387695, 21.890846252441406, -6.341148853302002, 36.270145416259766, 54.36653518676758, -1.9582769870758057, -0.06689947098493576, 76.5742416381836, -10.691291809082031, 12.601409912109375, 27.10208511352539, -64.60198974609375, 5.722619533538818, 20.20722770690918, 31.830833435058594, 26.487306594848633, 45.430442810058594, 6.1887054443359375, 13.821146011352539, 21.484142303466797, 4.976513385772705, 4.058466911315918, 22.873546600341797, -10.710277557373047, -44.37860107421875, -53.04830551147461, -5.498495578765869, -42.06519317626953, 33.27507781982422, 0.008209330961108208, 29.752012252807617, -5.092248439788818, 55.76034164428711, 43.751976013183594, 24.36856460571289, -9.800076484680176, -45.785457611083984, 43.87690734863281, 30.022058486938477, 21.794567108154297, 34.073326110839844, 5.481581211090088, 33.23978805541992, -27.226776123046875, 6.31116247177124, 16.343582153320312, -28.23598289489746, -24.990148544311523, 4.921527862548828, 40.87104797363281, -22.922157287597656, 23.819549560546875, -11.652215957641602, -45.4110107421875, 3.3689992427825928, -32.166831970214844, 18.710390090942383, -40.014869689941406, 5.671328544616699, 6.784780502319336, 15.490701675415039, 4.796518325805664, -51.17621612548828, 23.056894302368164, -37.752994537353516, 22.044628143310547, 48.44089126586914, -45.0871467590332, 6.7425923347473145, -2.338052988052368, 17.64339828491211, 5.445812702178955, 28.985788345336914, -14.28129768371582, -8.62509536743164, -35.00935363769531, 15.657686233520508, -72.6205825805664, -4.123968601226807, 28.160961151123047, -3.8094332218170166, 28.213035583496094, -12.199056625366211, -25.995838165283203, 11.80038833618164, -36.44313430786133, 50.73703384399414, -1.383459448814392, -41.632633209228516, 8.220597267150879, -11.498579978942871, 14.115962028503418, 17.91120147705078, -9.801177978515625, -16.47983169555664, -51.70674133300781, 34.60990905761719, -2.1742794513702393, 103.14485168457031, -14.843008995056152, -16.12067222595215, 19.129295349121094, 31.236230850219727, -6.03891134262085, 20.282777786254883, -42.844017028808594, 17.486053466796875, -56.24672317504883, -16.59366226196289, -2.98417067527771, 5.959434986114502, -14.422919273376465, -38.5517578125, -17.782453536987305, 22.20819091796875, -36.79530715942383, 32.44984436035156, 3.2231454849243164, -21.036394119262695, -21.461606979370117, -18.158586502075195, 26.32343101501465, 38.09453582763672, 46.432464599609375, -16.487123489379883, -50.037445068359375, -55.11930847167969, -3.9904353618621826, -53.831581115722656, -9.065315246582031, -16.136125564575195, 18.002822875976562, -7.900679111480713, 33.82354736328125, 37.39888000488281, -25.31671905517578, 2.020845413208008, -74.58948516845703, 28.828041076660156, 30.385272979736328, 20.92005157470703, 9.61223316192627, -16.787322998046875, -7.429667949676514, 39.161659240722656, -25.663803100585938, -1.962444543838501, 16.40231704711914, 84.04350280761719, -8.93439769744873, -2.5950767993927, -3.18485426902771, 20.578693389892578, 10.549752235412598, -58.29207229614258, -7.660561561584473, -9.998988151550293, 18.083824157714844, 10.184211730957031, -20.832687377929688, 47.897735595703125, -21.88469696044922, 8.958057403564453, -38.55247497558594, 27.830101013183594, -10.432184219360352, -1.1514657735824585, 87.88427734375, -9.384319305419922, 13.812411308288574, -76.80596923828125, -38.800201416015625, -9.779708862304688, 23.85171890258789, 4.693769454956055, 4.757492542266846, -7.851613521575928, 11.512053489685059, 6.8251118659973145, -10.293984413146973, 23.760679244995117, 53.802669525146484, -34.34571075439453, -15.02520751953125, -51.425941467285156, -30.502199172973633, -8.128602027893066, 47.568939208984375, 15.57554817199707, -6.798403263092041, 4.027426242828369, 2.047314405441284, 46.1219482421875, -1.2395284175872803, 11.431320190429688, 20.886322021484375, -19.214439392089844, 2.9547998905181885, 28.581634521484375, -5.768692970275879, -44.729522705078125, -33.43048095703125, -61.75249099731445, -14.519071578979492, -4.839145660400391, -16.99287223815918, 20.987552642822266, -22.170467376708984, 58.701087951660156, 9.631439208984375, 22.6939754486084, -12.901718139648438, -46.147987365722656, -38.15109634399414, -47.92141342163086, -22.497116088867188, -0.4058261513710022, -25.279733657836914, 10.10483455657959, 45.79212951660156, -10.44119644165039, -44.87250900268555, 20.106916427612305, 9.88021469116211, 18.49151039123535, -13.507413864135742, -1.18476402759552, 27.91474151611328, 35.89154052734375, 43.18989181518555, -1.2860541343688965, -48.76370620727539, -51.05792999267578, 2.653911590576172, 21.045969009399414, -43.438819885253906, 25.084732055664062, -4.894278049468994, 19.141002655029297, 22.994781494140625, 5.686486721038818, 20.309476852416992, 42.250545501708984, 10.297845840454102, 41.458282470703125, 30.038705825805664, -26.564990997314453, -10.277811050415039, 10.302775382995605, -59.179664611816406, -2.822080373764038, 27.893186569213867, 24.825002670288086, 24.23802947998047, 73.97874450683594, 18.72174835205078, 23.974231719970703, 30.55956268310547, -5.879957675933838, 46.60581588745117, 73.86172485351562, -50.02066421508789, 34.5858154296875, 28.166759490966797, -41.539154052734375, 10.718698501586914, 1.2324562072753906, -7.1734137535095215, -12.985546112060547, -9.897903442382812, 6.563033103942871, 4.197602272033691, -0.09668467938899994, 6.177926540374756, -10.980203628540039, 97.862060546875, -56.24388885498047, -13.080718040466309, 4.3256449699401855, 46.37614822387695, -36.67563247680664, -43.73306655883789, -20.56165885925293, 1.3713860511779785, 30.435665130615234, 1.3634653091430664, 43.523101806640625, -12.258362770080566, 11.51424503326416, -25.00851058959961, -12.267058372497559, 3.051180124282837, -6.17280912399292, 29.07440757751465, -6.226574420928955, 51.13190460205078, 0.9335517287254333, 35.49708557128906, -46.36647033691406, -0.6193175315856934, -17.758943557739258, 18.77766227722168, -26.450542449951172, -8.954362869262695, 2.9845638275146484, -14.60294246673584, 40.588233947753906, -17.197677612304688, 43.71211624145508, 73.02243041992188, -18.34242820739746, -29.036130905151367, -16.046171188354492, 24.191387176513672, 0.23498576879501343, 72.99051666259766, 105.6640625, -33.56894302368164, -39.83531188964844, 13.146349906921387, 30.166318893432617, 28.4741153717041, 21.46868133544922, -0.7886673212051392, -10.813558578491211, -30.88195037841797, 6.228184223175049, 30.176095962524414, 1.3600019216537476, -31.980710983276367, 36.84619903564453, -14.629595756530762, 26.246225357055664, -22.74078369140625, 14.747207641601562, -18.195741653442383, -23.645221710205078, 26.917089462280273, 22.083446502685547, 17.536697387695312, -27.686933517456055, 36.193721771240234, -41.34825897216797, 3.0988481044769287, 24.185903549194336, -43.46879577636719, 13.923041343688965, -3.325892448425293, -32.9372444152832, 33.740020751953125, -31.4591007232666, 32.81909942626953, 16.31462860107422, -36.793148040771484, -19.957191467285156, 29.90694236755371, -4.615759372711182, 10.446799278259277, -26.70549964904785, 37.39480209350586, -23.07637596130371, 2.6677608489990234, -38.35344696044922, 17.698200225830078, 12.313358306884766, 16.238435745239258, -30.419986724853516, 20.600086212158203, -10.189963340759277, -44.221885681152344, -19.632373809814453, 12.981979370117188, 25.68918800354004, 25.426206588745117, 24.434894561767578, -11.074333190917969, -21.705352783203125, 18.031394958496094, -18.440038681030273, 23.06842803955078, 30.637611389160156, 20.174480438232422, -2.363417625427246, 1.3781869411468506, 19.609262466430664, 15.625153541564941, -8.331615447998047, 38.79440689086914, -24.676715850830078, -68.32493591308594, 45.448974609375, 26.21339225769043, 10.408435821533203, 13.502777099609375, -26.175931930541992, -26.168819427490234, -29.888225555419922, 15.584176063537598, -26.472593307495117, 2.3630690574645996, -2.0393402576446533, 16.240764617919922, -1.6024978160858154, 29.55263900756836, -20.90082550048828, -47.514183044433594, -1.5896414518356323, -15.728547096252441, 52.854679107666016, 38.094940185546875, 5.520174503326416, -19.060136795043945, -25.55605697631836, 2.3205955028533936, -39.31988525390625, 57.374290466308594, 26.870943069458008, -39.54645538330078, 3.4961466789245605, -11.255914688110352, -38.54156494140625, -7.763073921203613, 35.88640594482422, -12.996631622314453, 30.144670486450195, -47.5218505859375, -1.2670921087265015, -49.82585906982422, 32.868377685546875, -3.5134668350219727, 11.024884223937988, -4.494327545166016, -1.296087384223938, -37.93864822387695, -27.393857955932617, 20.85190773010254, -37.14653015136719, 73.87479400634766, 31.394948959350586, -2.236525297164917, -73.41574096679688, 37.85439682006836, -22.27872657775879, 4.589256286621094, -29.69686508178711, 11.179667472839355, -45.096641540527344, -1.6933348178863525, 0.4578400254249573, -7.460495471954346, -5.581720352172852, -6.0754313468933105, 7.983950138092041, 38.127281188964844, -22.59464454650879, -11.685895919799805, 20.910686492919922, 12.613571166992188, -1.2570136785507202, -16.935426712036133, -27.736209869384766, 37.43082809448242, 34.684879302978516, -25.642881393432617, 14.46036148071289, 15.163457870483398, 10.178178787231445, -8.94691276550293, -16.858963012695312, -11.713234901428223, -1.931708574295044, -3.2753746509552, 26.856075286865234, 21.759862899780273, 17.48419189453125, 1.553910732269287, -26.824260711669922, 10.292851448059082, 3.2761476039886475, 30.75872230529785, -0.31291136145591736, 11.166838645935059, 29.221271514892578, -62.068878173828125, -49.19378662109375, -2.086092710494995, 41.17890167236328, 2.1543612480163574, 38.5764274597168, -28.14168930053711, -1.9649934768676758, -69.33954620361328, 24.071063995361328, 29.91167449951172, -25.247268676757812, -13.454168319702148, -44.21308135986328, 46.77058792114258, 7.3793745040893555, 24.183012008666992, -34.06380081176758, 11.177825927734375, 10.763190269470215, 46.69400405883789, 7.565222263336182, -49.161216735839844, -9.42851448059082, -54.64026641845703, 16.899776458740234, -26.333511352539062, 23.23885726928711, 36.27139663696289, -48.122371673583984, -24.33873748779297, -11.520431518554688, 16.845809936523438, -50.41829299926758, -9.416178703308105, -32.09442138671875, 12.301283836364746, -19.77142906188965, 11.712763786315918, 5.922928810119629, 43.185611724853516, -6.7198381423950195, -41.227882385253906, 29.84482192993164, -16.354183197021484, 4.4852800369262695, -64.8970947265625, 60.858177185058594, -34.862300872802734, 25.663436889648438, 23.150178909301758, -44.4819450378418, -41.085811614990234, -19.282638549804688, -15.73660659790039, 17.66936492919922, -24.065702438354492, -37.393253326416016, -25.649520874023438, -25.346975326538086, -5.685237407684326, -31.758895874023438, -47.07502746582031, -14.37259292602539, -13.141613960266113, 41.02301025390625, -18.425601959228516, -1.0655996799468994, 0.22266460955142975, 41.372825622558594, 53.39602279663086, 8.33884334564209, 21.335906982421875, -47.950889587402344, 8.1266450881958, -40.02627182006836, -16.773033142089844, -24.07941246032715, 58.9210205078125, -12.822244644165039, 2.4379310607910156, -17.09004020690918, 4.598710060119629, -46.21183776855469, 18.0860595703125, -8.158045768737793, 26.42110252380371, -23.582317352294922, -17.758766174316406, 0.029801294207572937, 7.262651443481445, -14.588044166564941, -31.486255645751953, -21.744922637939453, -8.11691665649414, -27.960113525390625, -23.95899772644043, -10.9210786819458, 61.55340576171875, 9.193531036376953, -9.491399765014648, -0.10019869357347488, -6.513235092163086, 1.124810814857483, -54.23634338378906, 0.8407289385795593, 36.287086486816406, 13.909817695617676, -15.399635314941406, -25.012840270996094, 13.727338790893555, -5.312813758850098, -23.743528366088867, -5.94379186630249, -6.863359451293945, 94.88737487792969, -29.201108932495117, -44.046207427978516, 31.661869049072266, 13.525773048400879, -7.074870586395264, 16.70077896118164, -17.625337600708008, 2.0878841876983643, -41.79477310180664, -49.817134857177734, 36.88325119018555, 34.315086364746094, 23.54281997680664, 29.97574234008789, -8.85118579864502, -28.084848403930664, 40.60488510131836, 16.474376678466797, -40.01177215576172, 64.95648956298828, -34.020172119140625, 10.143821716308594, -8.717819213867188, -25.50904083251953, -37.216957092285156, 33.20655822753906, 30.253429412841797, -37.5109977722168, 7.123808860778809, 31.503875732421875, 9.869115829467773, -11.91795825958252, 55.67833709716797, -22.233692169189453, 46.95553970336914, 35.3135871887207, 30.806922912597656, -12.876358032226562, 35.95972442626953, -53.99745178222656, 10.507268905639648, -10.958697319030762, 19.952186584472656, -60.98088836669922, -15.127930641174316, -30.1796932220459, -46.51487350463867, -8.603620529174805, 13.120800971984863, 45.99289321899414, 35.054039001464844, 22.640918731689453, -7.998539447784424, 18.04567527770996, 0.8130516409873962, -11.799073219299316, 18.551067352294922, 14.555635452270508, -30.436330795288086, -41.12762451171875, -45.22874450683594, 30.993301391601562, 12.010213851928711, -15.543996810913086, 1.801663875579834, -5.618455410003662, -14.597655296325684, -81.62985229492188, -29.628293991088867, -13.927238464355469, -24.5941104888916, 20.68000030517578, -3.577242136001587, -38.89371871948242, 30.51728057861328, -23.45728302001953, 30.220172882080078, 4.318844318389893, -7.886132717132568, 3.5533602237701416, 11.346902847290039, -8.510826110839844, -15.704144477844238, 10.908040046691895, -17.3055362701416, 58.560081481933594, -7.135922908782959, -14.166152000427246, -28.216344833374023, 70.93928527832031, -19.952102661132812, 8.029781341552734, -44.883113861083984, 10.380181312561035, -15.564984321594238, -7.987087249755859, 1.8216924667358398, 31.883729934692383, -15.863554954528809, -10.09683609008789, -10.21353530883789, -4.220609188079834, -55.653873443603516, 51.00739669799805, -56.912017822265625, -26.42646598815918, -16.405303955078125, 44.51520919799805, -18.17259407043457, 15.532790184020996, -8.889604568481445, -11.258008003234863, 7.981757164001465, 13.441228866577148, 16.88787269592285, -17.11229705810547, 10.98974609375, -33.057716369628906, 27.42644500732422, 5.03964376449585, 4.584782600402832, -6.339526653289795, 21.48619270324707, 7.075839042663574, 32.15019607543945, 1.417563557624817, -50.859310150146484, 23.465112686157227, -43.164554595947266, 19.039270401000977, -21.488819122314453, -8.597786903381348, -10.546100616455078, -15.32808780670166, -8.81322956085205, -8.811768531799316, -12.894949913024902, -38.93010711669922, 28.908157348632812, 16.1933536529541, 4.637226104736328, 50.86256790161133, 0.7157275676727295, -25.460098266601562, 0.513094961643219, 39.33405685424805, 14.38400650024414, 1.2635747194290161, -27.763065338134766, 27.128005981445312, -62.955020904541016, 26.510910034179688, 42.61558151245117, 22.747034072875977, -27.833227157592773, 15.41003131866455, 12.767034530639648, -11.26909351348877, -36.087989807128906, 9.028836250305176, 47.06918716430664, -29.421741485595703, 1.1304441690444946, -9.737740516662598, 35.055110931396484, -38.5567626953125, -2.0517501831054688, -37.10950469970703, -22.131057739257812, 10.841537475585938, 3.2091312408447266, 22.889415740966797, 29.9726505279541, -17.862411499023438, -42.559654235839844, 24.700332641601562, -33.97450256347656, -15.229745864868164, 27.429073333740234, -12.517285346984863, -8.876680374145508, -19.03772735595703, -46.60773849487305, -31.888107299804688, 0.3405213952064514, 17.364791870117188, 13.640238761901855, 3.913963794708252, 21.0093936920166, -8.7094144821167, -54.77880859375, 41.789180755615234, -19.15749740600586, -7.936283588409424, -24.4542179107666, 17.01604652404785, 14.826604843139648, -29.240036010742188, 26.59315299987793, 4.144522666931152, 9.113866806030273, 6.632023811340332, 16.70550537109375, -18.28192710876465, 36.56669616699219, 11.392898559570312, 25.367277145385742, 20.722549438476562, 51.3701057434082, -23.813743591308594, -12.303644180297852, -44.02330017089844, -13.005423545837402, -25.98687171936035, -42.933074951171875, 11.699695587158203, 27.905609130859375, 37.65950012207031, -26.465238571166992, 30.82317352294922, -30.898454666137695, -55.44186782836914, 25.663854598999023, -7.609846115112305, -69.03414916992188, 22.810592651367188, -12.20513916015625, 19.046741485595703, -14.448890686035156, -31.29867935180664, -16.647865295410156, -1.9835538864135742, -2.2918975353240967, 6.705304145812988, 92.7483139038086, 58.02558135986328, 2.023846387863159, -31.131637573242188, 13.464953422546387, -11.328800201416016, 18.975305557250977, 11.81672191619873, 31.227754592895508, -5.045741081237793, -69.92877197265625, -2.2085976600646973, -30.693700790405273, -4.320365905761719, -9.604644775390625, -20.830535888671875, -54.90071487426758, -22.670024871826172, 15.34582233428955, -19.475112915039062, -27.303062438964844, -9.61708927154541, -13.347213745117188, -45.876094818115234, 21.044960021972656, 3.767160415649414, -15.917617797851562, 9.109018325805664, 23.226470947265625, -0.6785117387771606, -4.387383937835693, 6.1653642654418945, 6.919356822967529, 12.008071899414062, -0.9438904523849487, 39.15851974487305, -41.60761260986328, 47.11159896850586, 13.582858085632324, 46.20915603637695, -18.44096565246582, -14.12093734741211, -31.09314727783203, 61.73766326904297, 11.292136192321777, -23.71912384033203, 28.55394744873047, 10.940735816955566, 10.408662796020508, 6.757651329040527, 21.654483795166016, 52.763877868652344, 14.251653671264648, -41.67538833618164, 19.54352569580078, -27.02090835571289, 3.251940965652466, -5.024836540222168, -13.918023109436035, -23.631010055541992, -30.85454750061035, -0.8412206768989563, 16.20389175415039, 58.159698486328125, -15.683473587036133, -61.98381042480469, -22.05206298828125, -0.15719544887542725, -0.34731125831604004, -98.65986633300781, -14.37601375579834, 45.6304817199707, -65.88319396972656, 8.858223915100098, 45.9531135559082, 50.809444427490234, 23.73310661315918, -14.482755661010742, 35.47623062133789, -78.75365447998047, -8.48607349395752, 31.18124771118164, -9.945488929748535, -4.994534492492676, 32.05223846435547, -13.905580520629883, -55.206939697265625, -19.005830764770508, 0.4621800184249878, 33.766136169433594, 4.041816234588623, -18.831096649169922, 2.195370674133301, -13.167256355285645, -12.137348175048828, -15.124765396118164, 34.407169342041016, 0.34314045310020447, 9.795214653015137, 45.7736701965332, -0.21944504976272583, 0.30704420804977417, 28.355276107788086, -23.889179229736328, 25.46314811706543, -9.968789100646973, -28.035825729370117, 15.972978591918945, -0.9987230896949768, -20.178869247436523, -49.74345016479492, 29.918071746826172, -19.789886474609375, 18.718351364135742, 14.586405754089355, 4.496893882751465, 9.41563606262207, 14.913492202758789, -16.939558029174805, -24.734506607055664, 32.795616149902344, -46.4514274597168, 22.435792922973633, 12.23210334777832, -17.003583908081055, 48.013580322265625, -32.61256790161133, -14.94053840637207, 17.37370491027832, -26.255651473999023, -13.169373512268066, -55.8956298828125, -9.506356239318848, 32.5912971496582, 36.01956558227539, -25.98277473449707, 16.646089553833008, 6.089410781860352, -5.109857082366943, 10.974748611450195, -46.68561935424805, -11.281420707702637, -17.687721252441406, 13.385671615600586, 69.85521697998047, -10.906119346618652, -14.494680404663086, 2.1097140312194824, -14.956101417541504, 0.45007798075675964, -1.0541611909866333, 6.18202543258667, 41.6218376159668, 41.4238166809082, -14.86624813079834, -40.25844955444336, -19.32568359375, -61.644710540771484, 10.429780960083008, 33.97018051147461, -21.756149291992188, -6.375339984893799, -0.41289976239204407, -39.91645050048828, 59.41526412963867, 60.16636276245117, -24.60090446472168, -2.799255132675171, -5.990140914916992, -33.782962799072266, 46.099090576171875, -59.418426513671875, 21.66554069519043, -4.289303302764893, -15.951412200927734, 5.565767765045166, 43.9451904296875, 10.629945755004883, -64.78398132324219, -33.89042282104492, 0.9981621503829956, -49.78514099121094, 24.0246524810791, -29.18231964111328, -20.59197425842285, -33.025245666503906, -5.011254787445068, -36.44995880126953, -36.352325439453125, 36.73851013183594, -43.28343200683594, -8.487725257873535, -37.970611572265625, 33.197052001953125, 38.75448226928711, -0.5534065365791321, 30.335290908813477, -2.932670831680298, -11.934172630310059, -33.300411224365234, 54.040470123291016, -17.128557205200195, 19.768266677856445, -11.509267807006836, -64.64761352539062, 1.3717522621154785, -69.69672393798828, 12.779382705688477, 14.704130172729492, -51.50547790527344, -2.4018609523773193, -55.66633224487305, 21.092941284179688, -6.468848705291748, -1.6867161989212036, 17.103822708129883, -44.33835983276367, -16.42333984375, -11.401449203491211, 45.47913360595703, 13.789666175842285, 12.672224044799805, 42.84910202026367, -8.916471481323242, -23.1237735748291, 33.40040969848633, 25.361177444458008, -50.38630676269531, -17.944799423217773, -31.992595672607422, 37.69418716430664, -73.59564208984375, -26.192235946655273, -33.17136001586914, 50.21061706542969, -13.618707656860352, 4.913539409637451, -62.20270919799805, 33.44984817504883, -41.492252349853516, -16.78317642211914, 50.11587142944336, -46.68076705932617, -3.5620405673980713, -1.208935022354126, 16.025165557861328, 39.91913986206055, 38.22685623168945, 6.442022800445557, -53.68555450439453, -64.00457763671875, -16.963769912719727, 8.445903778076172, -2.266007900238037, 5.601709365844727, 62.85870361328125, -16.791934967041016, -1.490765929222107, -2.947742462158203, 15.924274444580078, -24.44987678527832, 3.318464756011963, 13.744267463684082, 51.80107879638672, 28.618783950805664, 43.01451873779297, 1.017648458480835, 7.516736030578613, 20.87883949279785, -11.057724952697754, 20.985017776489258, -7.146881103515625, -29.749130249023438, 10.079992294311523, 34.230831146240234, 18.357311248779297, 10.01822566986084, -22.16222381591797, 14.575220108032227, 8.638809204101562, 34.34238052368164, 25.833044052124023, 16.54828453063965, -54.28989028930664, 26.62110710144043, 62.280879974365234, -9.952010154724121, -6.235229015350342, -41.53986740112305, 38.45928955078125, 6.212738990783691, 14.102310180664062, 11.773053169250488, 3.6713647842407227, -37.551429748535156, 19.176227569580078, -60.40263366699219, -6.342885971069336, -35.59574508666992, 37.37599563598633, 53.06694030761719, 22.6129093170166, -0.5035204887390137, 14.08228588104248, -34.82539367675781, 56.08924865722656, -8.445575714111328, -28.498271942138672, 24.859996795654297, 6.380611419677734, 18.551002502441406, 13.173446655273438, 11.100387573242188, 24.018156051635742, 8.370268821716309, 5.616281032562256, 41.3834114074707, 62.37777328491211, 24.8931941986084, 42.27035140991211, 6.183876037597656, 17.112289428710938, -46.89471435546875, -33.890953063964844, 7.688391208648682, -59.471397399902344, -66.7920150756836, -15.834065437316895, 12.802957534790039, 53.40098571777344, 26.67932891845703, -21.959463119506836, -77.54727935791016, 27.802440643310547, -15.97925090789795, -43.348838806152344, 50.044918060302734, 57.973838806152344, 6.519872665405273, 37.632850646972656, 24.597637176513672, -14.486645698547363, -10.131956100463867, -27.189830780029297, -36.333560943603516, -3.738269805908203, 27.246356964111328, 1.6788359880447388, 19.33852767944336, -14.537700653076172, -18.36553382873535, 22.7006893157959, 30.827186584472656, -66.77672576904297, -12.1235990524292, 18.59531593322754, 14.357095718383789, -20.44237518310547, -26.449525833129883, 63.31401062011719, -2.9755873680114746, -8.284955978393555, 20.313405990600586, 11.542163848876953, -21.94723129272461, 28.57961654663086, -71.03634643554688, -20.07708168029785, 9.672882080078125, 2.6523730754852295, 46.65812683105469, 8.791645050048828, -10.3478422164917, 2.9265897274017334, 57.96664810180664, 47.72892379760742, -36.5670051574707, -49.153892517089844, -16.138036727905273, 35.79090118408203, -15.596199989318848, -10.878695487976074, -23.620527267456055, -20.5549373626709, 53.14762496948242, 41.81153106689453, 17.98795509338379, -23.21504783630371, 29.495620727539062, 32.63124465942383, 17.57623863220215, -12.23232364654541, -31.485990524291992, -74.30821228027344, 8.48367977142334, 18.27733612060547, -43.359413146972656, -44.855201721191406, 51.11585998535156, -0.4111974537372589, 64.73798370361328, -10.819830894470215, -31.13912010192871, 33.26508712768555, -33.188636779785156, 4.451455116271973, 3.1072113513946533, -36.938873291015625, 26.61099624633789, 3.044755220413208, 34.27096939086914, -34.25593566894531, -9.058547973632812, 12.60086441040039, 29.725374221801758, -25.738391876220703, 1.8390302658081055, -28.462635040283203, 19.177759170532227, 6.909862041473389, -30.43334197998047, 18.49565315246582, -19.10965919494629, 105.31672668457031, -23.554672241210938, 23.073001861572266, -9.089645385742188, -47.43927764892578, 63.33924865722656, 6.317171096801758, 34.68312072753906, 17.6209716796875, 7.009450912475586, 0.9910386800765991, 25.982248306274414, -40.41349411010742, -0.7075397968292236, 20.650897979736328, -31.67783546447754, -24.472946166992188, 28.23736572265625, 7.906056880950928, 71.37471771240234, 30.658281326293945, -36.7159423828125, -23.40179443359375, 32.060157775878906, -40.49390411376953, 15.45096492767334, 6.524779796600342, -16.754558563232422, 20.758102416992188, 38.83556365966797, 24.077747344970703, -12.756810188293457, 24.944557189941406, 4.172267913818359, 4.6665191650390625, -20.53474998474121, -18.799137115478516, -19.967681884765625, -47.59349060058594, 30.796308517456055, 2.244272470474243, -10.1075439453125, 4.991509914398193, 51.114376068115234, -23.05361557006836, -21.711105346679688, 1.2834396362304688, 44.545753479003906, -5.6101226806640625, 2.4185056686401367, 20.97922134399414, -23.610862731933594, -22.012937545776367, 3.1012606620788574, -12.318867683410645, -14.502740859985352, -31.356998443603516, 36.30415725708008, 0.296246737241745, -10.988837242126465, -35.30659484863281, 33.986236572265625, 28.833486557006836, -3.297346830368042, 54.636314392089844, -47.202186584472656, 4.6566362380981445, -86.85761260986328, -7.958308219909668, -19.27020835876465, -7.585670471191406, 51.1202278137207, -7.506209373474121, -47.93097686767578, 21.740928649902344, 13.84471607208252, 69.99345397949219, -27.065093994140625, -41.69502258300781, 24.329893112182617, -15.019033432006836, 27.82273292541504, 40.013519287109375, -15.458863258361816, 18.610862731933594, -36.59742736816406, 9.032405853271484, 72.5926284790039, 19.17975425720215, -18.826805114746094, 15.348776817321777, -6.963467597961426, 6.450295448303223, -31.847412109375, -25.871959686279297, -14.194960594177246, 24.02321434020996, -22.22795867919922, 25.358339309692383, 19.93912696838379, 5.849668979644775, 40.05870056152344, 16.885276794433594, 81.2680435180664, -25.264476776123047, 7.90574312210083, 9.199751853942871, -4.54152250289917, -20.974079132080078, 18.274625778198242, -24.153392791748047, 25.073925018310547, -51.67881774902344, -13.534039497375488, -28.151424407958984, 12.552992820739746, 16.29010772705078, 70.42005157470703, 31.78398323059082, 4.372317314147949, 40.30100631713867, 1.9508562088012695, -0.11927220225334167, 44.033119201660156, -24.2283935546875, -71.11309051513672, -27.35672378540039, -3.955183982849121, -16.423259735107422, 3.0109059810638428, -74.58422088623047, -34.162315368652344, 0.11842550337314606, -45.874629974365234, -13.549212455749512, 3.5107367038726807, 1.9206979274749756, -12.536900520324707, 59.30207443237305, -53.086090087890625, 51.056243896484375, -43.6187858581543, -7.310225963592529, 24.267316818237305, 5.221107482910156, -20.37959098815918, -40.120445251464844, 9.165181159973145, -19.52068519592285, 30.698604583740234, -9.64085578918457, 27.203022003173828, -10.016754150390625, 29.056533813476562, 0.3640972971916199, 27.896759033203125, -33.56526565551758, -14.88386344909668, 11.40092658996582, -15.690293312072754, -13.954303741455078, -29.8762149810791, 25.1201229095459, 19.2698917388916, 36.656898498535156, 4.432227611541748, -15.065780639648438, -48.01017379760742, 3.8015034198760986, -14.52480411529541, 27.194669723510742, 6.651076316833496, -57.881622314453125, -16.779699325561523, -4.5784735679626465, -40.304969787597656, 23.558351516723633, -10.77325439453125, -118.94805908203125, 30.901065826416016, 15.477712631225586, -43.0425910949707, 11.672467231750488, 37.47737121582031, -36.165611267089844, 41.38954544067383, -18.72637939453125, -27.21832847595215, 10.580404281616211, 30.892925262451172, 12.296154022216797, -17.415315628051758, 23.261211395263672, -4.723883152008057, 2.7653534412384033, -1.3197468519210815, 6.654569149017334, 7.088486671447754, 49.06184005737305, -37.980342864990234, -15.909082412719727, -27.973724365234375, -38.316551208496094, -13.924256324768066, -19.6315860748291, -28.014314651489258, 40.748260498046875, 16.46394157409668, 23.621049880981445, -5.725883960723877, -19.95665740966797, -24.03526496887207, -6.9617414474487305, -6.843019485473633, 31.55892562866211, -43.20303726196289, -37.72494125366211, -18.674846649169922, 15.792464256286621, 30.082242965698242, 18.831571578979492, -43.51486587524414, -16.238277435302734, -8.886863708496094, 16.59585189819336, -6.464471817016602, 39.83179473876953, 10.245931625366211, -44.802127838134766, -14.780808448791504, 38.08633041381836, -34.77684783935547, 12.104146003723145, 51.35210037231445, 18.95828628540039, -54.342166900634766, 2.609973430633545, 13.506949424743652, 17.44094467163086, -14.431572914123535, -21.044443130493164, 24.9920654296875, 6.683748245239258, 38.40316390991211, 18.891138076782227, -14.285996437072754, 9.992484092712402, 27.021024703979492, -21.783029556274414, -22.280113220214844, 1.2898029088974, 46.50869369506836, 7.895977973937988, -14.14498233795166, 2.0238234996795654, 43.50113296508789, -36.03977584838867, 1.686034083366394, -19.5649356842041, 43.245670318603516, -21.22252655029297, 9.39236831665039, -20.428565979003906, -42.269195556640625, -1.9791309833526611, -0.7772007584571838, 34.17611312866211, -39.161659240722656, -25.500207901000977, 1.167237639427185, 25.859386444091797, 49.35146713256836, 42.327083587646484, 17.24406623840332, 7.867701530456543, 17.204172134399414, -29.34450340270996, -12.05664348602295, -39.112152099609375, 15.47000503540039, -10.72415542602539, -25.625749588012695, -42.860084533691406, 25.16933250427246, 15.54153823852539, -6.075671195983887, -21.851774215698242, 11.186981201171875, -22.798608779907227, 8.041998863220215, 35.0815315246582, 42.793006896972656, 35.850364685058594, -4.954448699951172, 10.773399353027344, 45.9333610534668, 4.769028186798096, -1.1780223846435547, -56.2374153137207, -11.105525970458984, 51.05052947998047, -40.08852767944336, 59.781429290771484, -7.924283504486084, 7.884380340576172, 5.835488319396973, -6.976663112640381, 6.785526752471924, 5.401560306549072, 43.04897689819336, -69.93260955810547, 29.729108810424805, -33.84922409057617, -22.83346176147461, -0.3388800024986267, 11.282280921936035, -29.026865005493164, 49.3990592956543, -34.0623893737793, -43.36353302001953, -17.744258880615234, -50.51542663574219, -3.4306018352508545, -25.669612884521484, 28.524551391601562, 6.178831100463867, 0.5684017539024353, -43.645137786865234, -6.852913856506348, -27.209136962890625, -26.78499984741211, 28.802505493164062, 25.75904655456543, 1.1100940704345703, -9.920525550842285, -9.127765655517578, -20.557415008544922, 4.922237396240234, -1.8118497133255005, 0.07053498178720474, 21.208118438720703, -38.16379165649414, -2.5706424713134766, -17.3018798828125, -9.320052146911621, 30.63845443725586, -17.440996170043945, -21.80714988708496, 9.848895072937012, 17.540708541870117, 53.99995422363281, 27.78907012939453, -6.503453731536865, 21.082639694213867, -24.362430572509766, -28.467517852783203, 30.20157241821289, -15.161125183105469, 16.625499725341797, 9.696938514709473, -21.878694534301758, 6.465813636779785, 26.868122100830078, -15.031771659851074, 49.40766143798828, 22.08819580078125, 40.03062057495117, 55.535728454589844, 14.121696472167969, -43.55788803100586, 13.181652069091797, 37.515037536621094, 6.740785121917725, -68.7270278930664, 13.367328643798828, -25.735158920288086, -29.161476135253906, -33.93937683105469, -25.59708023071289, 50.59489440917969, 51.47159957885742, 6.099995136260986, 15.055602073669434, -0.12834693491458893, -49.82606887817383, 31.508441925048828, -11.821338653564453, 70.09098815917969, 19.103981018066406, 22.340232849121094, 36.288082122802734, -27.655921936035156, 4.273748397827148, 27.71772003173828, 47.14680480957031, -29.129772186279297, -35.479373931884766, 50.10169982910156, -13.930842399597168, 3.05356764793396, 23.904834747314453, -5.431589126586914, -18.55695343017578, -21.953889846801758, -15.13241195678711, 28.341917037963867, -28.91686248779297, 35.385108947753906, 0.2999594509601593, -7.3018903732299805, -17.005229949951172, 10.809378623962402, -15.617026329040527, 2.505708932876587, -19.81656265258789, -47.53926086425781, 27.555383682250977, 8.75538158416748, 22.312828063964844, 7.276610374450684, -6.31942081451416, -17.93614387512207, -13.81550121307373, 9.587164878845215, -14.079812049865723, 44.251522064208984, 19.349262237548828, -17.22614097595215, -29.00665283203125, 9.48575496673584, -101.45825958251953, 17.987960815429688, 43.31034851074219, 26.67044448852539, -2.07045578956604, 38.635719299316406, 22.193702697753906, -11.572437286376953, -14.800661087036133, -24.52195167541504, 1.8437457084655762, 33.745567321777344, -24.081928253173828, 76.0998306274414, -27.494962692260742, -6.541515350341797, -20.795028686523438, -13.584100723266602, 33.05460739135742, -27.272729873657227, 17.760242462158203, -15.668996810913086, 27.891035079956055, 4.531651496887207, 30.006608963012695, -30.33867073059082, 0.9690952897071838, 41.891239166259766, 19.115882873535156, 22.282703399658203, 18.185266494750977, -14.372112274169922, 28.436128616333008, 13.729718208312988, -39.120704650878906, 53.66526412963867, -6.4332594871521, 40.29448318481445, 26.47938346862793, 9.555913925170898, -4.716838359832764, 49.877342224121094, 24.909290313720703, 13.273956298828125, -12.191723823547363, 2.989856481552124, -22.605653762817383, -13.55667781829834, -44.27265167236328, -29.303407669067383, 35.44050598144531, 21.80286979675293, -73.8738784790039, 5.431356430053711, 55.61960220336914, 76.79963684082031, 10.946677207946777, -39.43913650512695, 41.95469665527344, -57.898704528808594, 29.603469848632812, -19.619609832763672, -6.021109104156494, -30.284929275512695, 41.509925842285156, 0.12603875994682312, -28.0090274810791, -12.05055046081543, 64.03539276123047, 3.911074638366699, -42.589107513427734, 7.715421676635742, -16.66640281677246, 44.743064880371094, -85.44535064697266, 5.671470642089844, 23.52851104736328, -0.9509922862052917, -56.94274139404297, -25.678062438964844, 36.318267822265625, -40.690696716308594, -51.40544128417969, -44.863338470458984, -65.58604431152344, -10.615304946899414, 3.0882513523101807, 10.827930450439453, -36.168766021728516, -32.26243209838867, 20.27460289001465, -10.679010391235352, -8.108868598937988, 36.360198974609375, -31.710405349731445, -38.5989990234375, 1.787562608718872, 0.12743151187896729, 2.4349870681762695, 22.374662399291992, -24.830835342407227, 17.9776668548584, -41.151466369628906, 28.126338958740234, 36.22553253173828, -4.1204681396484375, 0.3003658652305603, -36.45197296142578, 0.7632452249526978, 6.258609294891357, 5.797947406768799, 29.995668411254883, -17.229875564575195, -29.96201515197754, 19.981454849243164, 29.281644821166992, -11.368253707885742, 24.912273406982422, 11.353004455566406, -7.08574914932251, -35.324859619140625, 56.19636917114258, 8.340232849121094, 20.0562686920166, -25.479415893554688, -29.753305435180664, 38.6507568359375, 12.255647659301758, 20.667524337768555, 6.666426181793213, -5.7545881271362305, -30.405338287353516, -7.618605613708496, -1.317691683769226, 33.2126579284668, 43.374298095703125, -23.03841781616211, 27.00901985168457, 9.18600845336914, -2.233250141143799, 53.21406173706055, -10.096054077148438, -3.093710422515869, 24.23661994934082, -5.276391506195068, 53.08163070678711, -16.414226531982422, 0.32846149802207947, -22.080425262451172, 2.005821943283081, 29.532676696777344, -16.6795654296875, 27.687341690063477, 0.47772449254989624, 5.5754475593566895, -24.917095184326172, 18.366455078125, -21.39177703857422, -33.537540435791016, -52.77788162231445, 56.796993255615234, 1.426247000694275, 8.086844444274902, 41.49401092529297, -32.28608703613281, 84.12444305419922, -23.142982482910156, -19.19766616821289, -34.81002426147461, 12.127952575683594, 19.36688995361328, 41.53916931152344, 8.25688648223877, -59.206729888916016, -3.4796671867370605, 59.48282241821289, -31.944595336914062, 18.694713592529297, 4.811857223510742, 37.397491455078125, -10.103318214416504, 8.508299827575684, 10.506251335144043, -18.954055786132812, -16.98301124572754, -11.407363891601562, -40.882877349853516, -16.22271728515625, 30.160245895385742, -28.667993545532227, 16.885408401489258, 35.76835632324219, 16.72926139831543, -4.630145072937012, 55.9638557434082, 8.672216415405273, 8.538721084594727, -51.540740966796875, -25.77901840209961, -0.630195677280426, 27.667232513427734, 2.782102108001709, 13.410008430480957, 17.83659553527832, -10.851737976074219, 30.699478149414062, -15.882014274597168, 14.441278457641602, -5.596010684967041, -55.02796173095703, -5.40156888961792, 37.9001350402832, 30.71148681640625, 14.609621047973633, -1.0671298503875732, -9.18587875366211, -3.937527894973755, -71.46821594238281, -15.608203887939453, 9.890291213989258, -7.675153732299805, -32.03065872192383, 52.687313079833984, 8.228886604309082, -12.0245361328125, -38.057159423828125, 6.802901268005371, 5.259655475616455, -85.05121612548828, 13.229096412658691, 18.056814193725586, 30.32166862487793, -62.53587341308594, 24.31583023071289, 2.122018575668335, 8.594809532165527, -46.26300048828125, -6.29630708694458, 32.63118362426758, 2.5487020015716553, 10.374906539916992, -1.8182209730148315, -12.232095718383789, 62.85512924194336, 0.46019697189331055, 10.981771469116211, 19.448137283325195, 46.755245208740234, 9.467476844787598, 12.1339750289917, -25.76007080078125, 35.93137741088867, 46.69801712036133, 3.456535816192627, -16.62242889404297, 43.871456146240234, 66.1906967163086, -14.004322052001953, -16.658946990966797, -2.7454159259796143, 42.26738357543945, -8.171528816223145, -41.328338623046875, -14.856328010559082, 10.158184051513672, 22.476587295532227, -34.85487747192383, -11.094941139221191, -23.119579315185547, 17.601518630981445, -37.81379699707031, 49.69591522216797, -7.97824239730835, -51.37653350830078, 27.135517120361328, -50.21609115600586, 19.05752182006836, -23.068727493286133, -6.068957328796387, -44.432525634765625, -0.9106948375701904, 20.53845977783203, 2.4351460933685303, -15.553092956542969, -38.21485900878906, -12.539560317993164, 22.611282348632812, 37.079803466796875, -10.82655143737793, -59.05242919921875, -3.0957112312316895, -72.01407623291016, 40.06515884399414, 26.032684326171875, -47.377235412597656, -14.441161155700684, -8.399513244628906, -39.22123336791992, -5.389922142028809, -9.960521697998047, 43.546730041503906, 12.092954635620117, -2.3118538856506348, 35.46246337890625, -8.542258262634277, 67.41019439697266, 23.065704345703125, -1.3251115083694458, -1.1465861797332764, 18.823408126831055, 7.236114025115967, -34.77447509765625, -55.74905776977539, -10.711250305175781, -28.784080505371094, 38.89832305908203, -21.76849365234375, 35.304832458496094, -73.09226989746094, -4.412031650543213, 53.76718521118164, -10.990799903869629, 31.746809005737305, -23.497285842895508, 24.555740356445312, 22.766239166259766, -8.781356811523438, -69.06600189208984, 4.334229946136475, 73.15937042236328, -16.799890518188477, 56.81490707397461, -21.166820526123047, 38.11659622192383, 12.439370155334473, -19.592893600463867, -16.91141700744629, -4.955608367919922, -29.4252872467041, 36.61233139038086, -5.315252304077148, -9.10466194152832, -10.713716506958008, -2.193877935409546, -2.8262412548065186, -24.02379035949707, 8.142020225524902, -13.379777908325195, -48.87242889404297, -0.9127820730209351, 42.14596176147461, -51.62172317504883, 12.891475677490234, 35.12262725830078, 70.93707275390625, 9.079370498657227, 13.163047790527344, 46.716007232666016, -9.302640914916992, -5.981407642364502, 2.263319492340088, -54.38132095336914, -2.222654342651367, -10.201168060302734, 9.062637329101562, 11.121650695800781, -3.473198652267456, -40.194183349609375, -9.338720321655273, -26.748950958251953, -46.2535514831543, 45.0661506652832, 9.77480411529541, 4.554952621459961, -45.77155685424805, -48.91643524169922, 26.04240608215332, -22.01321029663086, 16.433809280395508, -7.48124885559082, -6.219464302062988, -30.374675750732422, 16.723613739013672, 5.826491355895996, -39.207454681396484, -23.990999221801758, -1.174674391746521, 18.65750503540039, 5.829523086547852, -79.1555404663086, -12.617279052734375, -34.30022048950195, -34.10074996948242, -12.748106002807617, 1.436579704284668, -49.63689422607422, 45.00973892211914, -19.311426162719727, 23.627758026123047, 14.947654724121094, -30.955982208251953, 3.103242874145508, -14.482324600219727, -15.49028491973877, 44.599552154541016, -18.42713165283203, -4.815189361572266, -13.617820739746094, -31.805906295776367, 11.791675567626953, -5.338932037353516, -30.318557739257812, 32.139427185058594, -0.9864510893821716, -49.21489334106445, 30.809505462646484, 44.633628845214844, -69.76815032958984, -2.8266775608062744, -20.81548500061035, -31.865093231201172, -16.959787368774414, 26.12279510498047, 32.029632568359375, -3.0466973781585693, 20.455833435058594, 43.52417755126953, -16.698469161987305, 25.825220108032227, -23.811969757080078, 23.4991397857666, 47.35232162475586, -15.63698673248291, 7.427439212799072, -10.710542678833008, 33.27388381958008, -32.30938720703125, -24.938234329223633, -45.954219818115234, 25.456859588623047, 53.28569030761719, -22.005699157714844, 18.572689056396484, 10.214186668395996, -28.2069034576416, -3.11688494682312, 8.842057228088379, -22.999544143676758, -17.459665298461914, 20.181615829467773, -32.046287536621094, 16.572921752929688, 7.948526382446289, -38.801509857177734, -1.2824331521987915, 35.657596588134766, -11.035436630249023, -43.546878814697266, -53.52371597290039, 2.2396371364593506, -9.206793785095215, -19.443269729614258, 6.005033493041992, 8.063413619995117, 22.633703231811523, -18.58962631225586, 33.513023376464844, -29.87575912475586, -3.2044732570648193, 15.009931564331055, -6.629897117614746, 7.982985496520996, -20.580034255981445, 4.562582492828369, 46.62984848022461, 25.279247283935547, -42.512977600097656, -26.472185134887695, 6.709238052368164, -14.714622497558594, -14.398358345031738, 5.097537040710449, -21.99468994140625, -27.93084144592285, -50.704132080078125, -59.76461410522461, 30.619075775146484, 15.163070678710938, 40.83734893798828, 29.387453079223633, -58.54480743408203, 40.58680725097656, 14.892632484436035, -37.54021072387695, 9.690733909606934, -30.170116424560547, 24.14328956604004, -53.08543014526367, -43.80369186401367, 3.6786723136901855, -49.19091033935547, -47.44023895263672, 3.232550859451294, 15.137856483459473, -33.0538330078125, 39.53115463256836, 44.32704162597656, -5.803711414337158, 10.852622985839844, -11.84687614440918, -11.367902755737305, 13.161367416381836, -13.344632148742676, -13.219868659973145, -11.421926498413086, -10.526508331298828, -27.023075103759766, -3.88991379737854, -22.90617561340332, -24.870864868164062, -13.258064270019531, 42.441131591796875, 19.8499813079834, 8.680391311645508, -5.276516914367676, 0.882489800453186, -9.1471586227417, -16.29548454284668, 27.049673080444336, 26.041765213012695, -29.11067008972168, -40.29043960571289, 39.48970031738281, -21.239219665527344, -34.99300003051758, 8.246966361999512, 33.064701080322266, 36.90589141845703, -4.647298812866211, 22.658781051635742, -27.201509475708008, -23.273151397705078, -34.47026062011719, 22.167133331298828, -6.451584815979004, -25.32322120666504, 3.7927346229553223, 23.975177764892578, -96.94364929199219, 19.481576919555664, -11.46164321899414, 40.477352142333984, -12.50003433227539, 20.178733825683594, 11.47136116027832, 16.071245193481445, -24.432016372680664, 45.20137023925781, 0.4061166048049927, -28.178014755249023, 0.09846529364585876, 53.69624710083008, -1.2989163398742676, 14.613759994506836, -31.323848724365234, -26.071727752685547, -20.52707862854004, -5.183660984039307, -9.478445053100586, 14.669437408447266, 34.71453857421875, -20.401243209838867, 19.423297882080078, -28.974903106689453, 1.660184621810913, 0.27234628796577454, 9.398813247680664, -43.35208511352539, 17.443769454956055, 42.518497467041016, -41.27124786376953, 66.13975524902344, -30.89106559753418, 21.001264572143555, 18.015853881835938, -26.871013641357422, -29.81952476501465, 53.31460189819336, 24.831817626953125, 26.861454010009766, -14.727624893188477, 51.12181854248047, -12.448012351989746, -8.193916320800781, 37.35286331176758, 0.7211095690727234, 33.38159942626953, 1.697637915611267, 19.905193328857422, 33.675315856933594, -10.064593315124512, 16.221939086914062, 37.768798828125, -21.497560501098633, 34.94096374511719, 31.190561294555664, 47.67662048339844, 12.079917907714844, -1.1630866527557373, -13.665177345275879, 9.393268585205078, -27.9866886138916, -1.1547348499298096, 15.196352005004883, 66.47193908691406, 5.018240451812744, -15.606986045837402, 42.25010299682617, -11.371003150939941, -21.522918701171875, -12.520448684692383, -36.600948333740234, -4.242498397827148, 22.288246154785156, -33.110130310058594, -57.168701171875, 8.729757308959961, -28.10349464416504, 25.94226837158203, -12.27015209197998, 77.67908477783203, 27.063316345214844, 15.5, 24.810338973999023, 52.747947692871094, -50.060245513916016, 18.760295867919922, -17.20261001586914, 19.69393539428711, -11.211700439453125, -11.426431655883789, -3.3721022605895996, 21.771299362182617, 23.143781661987305, 33.698936462402344, 39.15032958984375, 10.771759033203125, -49.89546203613281, -40.89420700073242, -2.0989043712615967, -80.26296997070312, -38.81159973144531, -40.74983596801758, -6.8802056312561035, -4.2199225425720215, -60.550315856933594, 40.7685432434082, 16.887895584106445, 18.39453887939453, -31.428071975708008, 9.915833473205566, -30.799558639526367, 6.782228946685791, 9.243156433105469, 5.963152885437012, 22.049840927124023, 24.159725189208984, -37.789573669433594, 3.492107629776001, 0.28086841106414795, 10.531200408935547, -11.268921852111816, 40.753936767578125, -0.14858558773994446, -2.8387367725372314, 14.358709335327148, -9.854741096496582, -5.553549766540527, 14.1635103225708, -32.5766487121582, 43.035789489746094, 20.277883529663086, -6.0414228439331055, -69.88668060302734, 20.463138580322266, -32.13914108276367, 15.773536682128906, 55.65270233154297, 77.973388671875, 27.142568588256836, 44.851112365722656, -0.15676721930503845, -45.320194244384766, 18.913543701171875, -6.789809703826904, 11.614887237548828, 20.201087951660156, -21.59461212158203, 0.7362760305404663, -5.504433631896973, 24.153751373291016, -41.25407028198242, 25.49863624572754, 54.39134979248047, -42.81513977050781, -6.962447166442871, -20.33551597595215, -8.436834335327148, -22.06218147277832, 2.583742141723633, -9.20593547821045, 23.977834701538086, -41.150978088378906, -41.72019958496094, 21.71610450744629, -6.5091776847839355, -21.873329162597656, -54.085365295410156, 1.1206846237182617, 8.122625350952148, -69.39117431640625, -36.724708557128906, -7.440035820007324, -19.363521575927734, -35.29661560058594, 1.8148846626281738, -20.194580078125, -8.931597709655762, -55.7988395690918, -28.89874839782715, 18.622528076171875, -44.01801681518555, 24.908876419067383, 16.68628692626953, 54.210052490234375, -33.43811798095703, -1.6273819208145142, 43.84749984741211, -0.7853540778160095, 17.468891143798828, -3.658019781112671, -3.1210174560546875, 18.29295539855957, -27.652376174926758, 48.042598724365234, 16.682180404663086, 25.532201766967773, -74.20104217529297, 25.491329193115234, 3.139056444168091, 27.900653839111328, -2.7323436737060547, -23.011133193969727, -13.063037872314453, 31.641761779785156, -35.85954284667969, 4.289346694946289, -16.2326602935791, -15.847929000854492, 10.749303817749023, 7.2275824546813965, 18.95367431640625, 22.800464630126953, -16.04169464111328, -19.60957145690918, -37.391971588134766, -59.29502487182617, 26.793376922607422, -18.257333755493164, -45.66984176635742, -39.91018295288086, -7.327240467071533, 16.13563346862793, -35.48440933227539, -31.2169246673584, 53.36851119995117, 0.448409765958786, -51.458133697509766, 32.707462310791016, -5.7036614418029785, 12.145997047424316, 39.660850524902344, -16.534608840942383, -1.642270803451538, -21.693195343017578, 26.520523071289062, -2.1368319988250732, -47.711875915527344, -14.97570514678955, 11.087018966674805, -3.1415231227874756, -8.303726196289062, 30.64558982849121, -6.565178871154785, 10.899103164672852, -41.350494384765625, -51.4702262878418, -29.198915481567383, -25.600149154663086, -44.407745361328125, -5.212913513183594, 50.74467849731445, 22.328737258911133, -3.0809249877929688, -15.72111701965332, -56.006004333496094, 0.9407932162284851, 13.956074714660645, -27.215600967407227, -4.638645648956299, -36.62971115112305, -19.02666473388672, 10.852747917175293, 16.133087158203125, 10.004262924194336, -5.094013690948486, 7.044171333312988, 28.976802825927734, 4.629415035247803, 26.97504425048828, -17.8656005859375, -1.540271282196045, -11.570868492126465, 39.20832443237305, 31.157533645629883, 19.174190521240234, -24.611867904663086, -12.486412048339844, 25.457847595214844, 14.080310821533203, 28.259187698364258, 30.034772872924805, 55.45796585083008, -0.9537710547447205, 10.997281074523926, 6.457584381103516, -14.809273719787598, -16.588455200195312, -22.3957462310791, 1.0921530723571777, 27.018171310424805, 45.58088302612305, 76.41282653808594, -15.73055362701416, -18.557872772216797, 56.443424224853516, -29.061086654663086, -14.43405532836914, 46.104713439941406, 28.679141998291016, 2.2773663997650146, -3.3635365962982178, 28.780017852783203, 5.182225227355957, -73.30638885498047, 18.18299102783203, 10.98747444152832, 34.704246520996094, -16.33325958251953, -24.537744522094727, -5.05258321762085, 36.653724670410156, -45.056514739990234, 0.8264258503913879, 16.655797958374023, 3.529566526412964, -22.412961959838867, 4.360276699066162, 1.010392427444458, -5.574038505554199, 37.803184509277344, -15.079773902893066, -16.52544593811035, -3.043484687805176, 36.62434768676758, -4.599405288696289, 27.970046997070312, 22.719989776611328, -35.891414642333984, -0.14971378445625305, 3.82843279838562, 7.312140464782715, -12.678179740905762, 42.18302536010742, 20.49785041809082, -37.524436950683594, -28.01796531677246, 12.21360969543457, -27.739978790283203, 7.348398685455322, 53.70354461669922, -30.783905029296875, 28.808610916137695, -19.603525161743164, -23.379003524780273, -8.496132850646973, -11.305486679077148, -3.1317644119262695, -20.313650131225586, 16.746522903442383, -27.00092887878418, -13.336939811706543, 12.68476676940918, 58.14585494995117, 7.216307163238525, 31.23863983154297, -1.966888666152954, -28.175487518310547, -13.003148078918457, -56.32943344116211, 9.587179183959961, -1.8725019693374634, 17.40568733215332, 1.557527780532837, -4.086796760559082, -60.15015411376953, 31.42448616027832, 11.84994125366211, -41.95449447631836, 14.548126220703125, 13.04285717010498, 26.25665283203125, -53.31311798095703, -1.4077147245407104, 45.160186767578125, 15.149259567260742, -27.226015090942383, 15.63217544555664, 2.2065696716308594, -27.91875648498535, -52.16448211669922, -24.74252700805664, -31.044490814208984, 13.511273384094238, 16.30712127685547, 4.224513053894043, -14.526538848876953, 35.617591857910156, -56.07854461669922, -23.79840660095215, 12.463509559631348, -25.538843154907227, 46.235958099365234, -57.12290573120117, -24.830045700073242, -22.583234786987305, 29.12561798095703, 35.83539581298828, -60.917545318603516, -36.87767028808594, 11.535661697387695, 26.993803024291992, -14.146294593811035, 5.083825588226318, 0.9991999864578247, 15.551149368286133, -8.079397201538086, -37.707725524902344, 55.0893440246582, 1.339864730834961, -65.6190185546875, 39.7773551940918, -4.571810722351074, 41.90145492553711, -15.484496116638184, -5.152442932128906, 33.61865234375, -5.839159965515137, -35.2704963684082, -12.09050464630127, -25.11925506591797, 18.40810203552246, -36.28536605834961, 46.79989242553711, 14.665778160095215, 3.1052374839782715, -1.500361442565918, 1.9630415439605713, 11.268721580505371, -44.531734466552734, -16.162254333496094, -37.62989044189453, -31.059844970703125, 22.519515991210938, -44.969913482666016, -25.667640686035156, -41.736106872558594, -53.5233039855957, -15.20596694946289, 19.249126434326172, -23.499412536621094, 4.207112789154053, -27.628376007080078, 19.41596221923828, 13.173311233520508, -12.660636901855469, 6.619071006774902, -6.689995288848877, 31.493017196655273, -24.432315826416016, 3.2161197662353516, -6.018987655639648, 9.429373741149902, -12.58608627319336, 30.24123764038086, 22.022924423217773, -29.28586196899414, -8.51303768157959, -8.840303421020508, -12.246244430541992, -36.538211822509766, 14.66523265838623, -26.47601890563965, -19.204082489013672, 21.370603561401367, 49.04167556762695, 0.7599825263023376, -2.812678098678589, -47.63695526123047, -0.6386460661888123, 29.928306579589844, -20.765905380249023, 39.47743606567383, -3.5089166164398193, -29.256298065185547, -24.971376419067383, 32.42903137207031, 9.79144287109375, -5.270422458648682, 4.128381729125977, -6.831416606903076, 11.477705955505371, -19.909500122070312, -0.20785582065582275, 14.699305534362793, -41.74803924560547, 16.810455322265625, 27.587404251098633, 24.796911239624023, 7.4632391929626465, -58.9144287109375, -34.81300354003906, 26.64778709411621, 54.763851165771484, 28.750696182250977, -32.21705627441406, 11.720847129821777, -40.53202438354492, 5.292169094085693, 22.905109405517578, 50.20219039916992, 33.2750358581543, -14.186673164367676, -48.886756896972656, -0.09671123325824738, -15.7073392868042, -4.592812538146973, 48.649173736572266, 39.86835861206055, 11.171083450317383, -15.311368942260742, -28.383358001708984, -6.297244548797607, -60.50939178466797, -40.00218963623047, 26.5093994140625, 92.18822479248047, 28.496644973754883, 20.99410057067871, 26.33123207092285, -7.755420684814453, 5.578618049621582, -11.989363670349121, 0.16267967224121094, -8.276083946228027, -67.62187957763672, 44.84964370727539, -16.432029724121094, 5.7907257080078125, 45.91212844848633, -71.94851684570312, -1.914026141166687, 29.436607360839844, 1.2830660343170166, 13.991852760314941, 31.853137969970703, 3.879979372024536, -27.503177642822266, -29.843944549560547, -0.8647845983505249, -12.684136390686035, 32.17013168334961, 22.2640438079834, 13.621583938598633, 15.019177436828613, -5.365592956542969, 8.324437141418457, 35.4703369140625, -20.523439407348633, 29.020156860351562, -9.423852920532227, -27.105287551879883, 14.478018760681152, -5.117786884307861, 33.75871658325195, -8.762701988220215, -55.161800384521484, 36.91773223876953, -2.923173427581787, -3.7134222984313965, 51.352760314941406, 61.1987190246582, -0.7332208156585693, -17.11882781982422, -16.121891021728516, 5.8871541023254395, 6.9591755867004395, 49.7446403503418, 17.66759490966797, -5.073501110076904, 11.111327171325684, -12.153301239013672, 40.605796813964844, -10.05143928527832, 73.98446655273438, 16.614192962646484, 41.393531799316406, 8.902320861816406, -19.786460876464844, -39.2368049621582, -8.093841552734375, -13.45655632019043, 34.626502990722656, -58.5407829284668, -18.872760772705078, 18.738685607910156, 23.294694900512695, -16.848390579223633, -13.843795776367188, -24.647653579711914, -10.415199279785156, -6.279577255249023, 20.35908317565918, 33.08833694458008, -10.014060020446777, 24.174915313720703, 23.567489624023438, 20.84299659729004, -53.53137969970703, -29.05533218383789, 32.33396911621094, -69.3004150390625, -49.931148529052734, -35.046592712402344, -50.833858489990234, 28.222232818603516, -4.509791851043701, 11.071090698242188, 21.97414207458496, 8.213472366333008, -19.477203369140625, 1.879903793334961, 0.6236191391944885, 4.384194374084473, -31.004959106445312, 43.479068756103516, -21.440227508544922, 8.381641387939453, -56.399742126464844, 18.519731521606445, -23.926755905151367, -7.624861240386963, -16.316837310791016, -14.7557954788208, -29.94442367553711, 4.653581142425537, -44.29237365722656, -46.306461334228516, 14.005236625671387, -4.66319465637207, -9.147683143615723, -19.542909622192383, 6.424307823181152, 7.589715480804443, -9.903735160827637, -19.071611404418945, 16.486347198486328, -54.230072021484375, 1.2987620830535889, -10.950214385986328, 21.07612419128418, -11.52013111114502, -74.64530944824219, -27.058910369873047, -16.450244903564453, 38.88691329956055, -58.56477737426758, -17.388954162597656, 2.969102144241333, -18.18324851989746, 17.194263458251953, 16.297054290771484, 13.782000541687012, 22.96451187133789, 26.173280715942383, -25.627775192260742, -7.331173419952393, 38.96000671386719, -27.154319763183594, 24.338520050048828, 15.058754920959473, -27.122461318969727, 58.67657470703125, 2.2956881523132324, 39.51431655883789, -31.0579776763916, 68.03072357177734, -5.077828407287598, 3.8881618976593018, -65.92384338378906, -0.7898028492927551, -31.753389358520508, -0.24916204810142517, -26.12610626220703, 5.246144771575928, -45.13970184326172, 8.102444648742676, 31.625890731811523, -8.003662109375, 54.2281494140625, -50.6468505859375, 20.889129638671875, 21.288909912109375, -61.818809509277344, 8.724381446838379, -29.71200180053711, 41.22616958618164, -78.34010314941406, 22.609546661376953, -22.209556579589844, 4.966239929199219, 7.817788124084473, 29.281021118164062, 48.77298355102539, 55.52143096923828, 36.35892105102539, -45.48480987548828, 57.55699920654297, -20.80007553100586, -50.67418670654297, 27.357210159301758, -18.56113624572754, -6.118281364440918, 19.352699279785156, -32.334712982177734, 41.40521240234375, 42.501644134521484, -54.44197463989258, 41.81808090209961, 10.114633560180664, -4.1641364097595215, 9.094137191772461, 33.40641784667969, 15.380791664123535, -23.267980575561523, -8.525242805480957, 25.488813400268555, 18.050294876098633, -36.829776763916016, 13.467147827148438, -53.34425354003906, 40.16396713256836, 34.57624435424805, 17.867717742919922, 0.5277529954910278, -17.83888816833496, 7.371458053588867, -8.794787406921387, 54.58782958984375, -10.969634056091309, 43.2191047668457, 70.68952941894531, -20.44845962524414, 34.112003326416016, -55.39177703857422, 3.4123764038085938, -34.34794235229492, 12.958234786987305, -75.89372253417969, -5.123180389404297, 61.565879821777344, -0.8720216751098633, 50.94319152832031, 7.823696136474609, 1.2732222080230713, -20.768123626708984, 50.923580169677734, 7.859521389007568, -18.459821701049805, 13.655922889709473, 6.913986682891846, 21.10057830810547, 8.390257835388184, -15.721240997314453, -1.3835355043411255, -12.95456314086914, -11.744431495666504, 79.65706634521484, -26.371824264526367, -5.848409652709961, 35.65508270263672, -56.38994598388672, 2.84649920463562, 44.01862716674805, 13.024947166442871, -25.304393768310547, 6.232673168182373, 23.894912719726562, 39.48470687866211, 9.663712501525879, -0.7481220364570618, 33.60700225830078, 20.50826072692871, 2.883546829223633, 25.093738555908203, 25.52681541442871, -6.313739776611328, -29.044527053833008, -33.24523162841797, -0.32708197832107544, -3.696913003921509, -27.886581420898438, 49.24786376953125, -6.652379035949707, 11.48717975616455, 6.174018859863281, 4.15742301940918, 18.361164093017578, 11.44825553894043, 32.36501693725586, 3.227125883102417, -0.626936674118042, -43.49319839477539, -34.75761795043945, 24.291845321655273, -19.045730590820312, 26.43178367614746, 25.91872215270996, -8.639786720275879, -16.30267333984375, -73.71446228027344, -18.033422470092773, 31.98279571533203, -20.276567459106445, 50.48674011230469, 11.23295783996582, 20.22711944580078, -46.40220260620117, 6.992278575897217, 76.61715698242188, 17.7623348236084, -10.871373176574707, -20.76169204711914, -30.414472579956055, 17.804466247558594, 3.3029518127441406, -47.16373062133789, -14.205991744995117, 12.854718208312988, -73.93305206298828, -6.30582857131958, 54.30438995361328, -23.074464797973633, 13.475696563720703, 32.68307876586914, 16.40731430053711, -0.8469801545143127, 14.154165267944336, 38.24135208129883, 2.3144443035125732, -15.780378341674805, 61.197086334228516, -13.410847663879395, -18.194808959960938, 5.590647220611572, -21.801067352294922, -3.4822750091552734, -0.7427595853805542, -62.45049285888672, 13.226086616516113, -17.972463607788086, -6.1812896728515625, -18.033924102783203, 5.799630641937256, -21.108728408813477, -32.77752685546875, -17.774826049804688, -8.896462440490723, 5.918873310089111, -52.82750701904297, 6.606382846832275, 1.753317952156067, 26.472108840942383, 11.850918769836426, 20.428224563598633, 20.93768882751465, -19.365266799926758, 13.858970642089844, -16.58745765686035, 10.934889793395996, -39.00457763671875, -29.37261962890625, -17.954370498657227, 12.763870239257812, 74.07140350341797, -25.454687118530273, 33.04792022705078, 69.60404968261719, 36.22914505004883, -9.0540132522583, -5.7222981452941895, 12.503365516662598, 78.27444458007812, 15.091329574584961, -5.568584442138672, 11.902068138122559, -45.89507293701172, 37.1457405090332, 29.534879684448242, -34.49211120605469, -10.434494018554688, 24.471132278442383, -25.04311180114746, 9.884281158447266, -0.34271594882011414, 34.02922821044922, 18.662328720092773, 0.5055677890777588, -15.748275756835938, 18.790552139282227, -17.803564071655273, 1.3866486549377441, -5.702437400817871, 23.33755111694336, -7.145149230957031, -46.46300506591797, 1.2019566297531128, -30.945724487304688, 5.621657848358154, -24.546857833862305, 28.80644416809082, -31.660974502563477, 21.99701499938965, 74.46153259277344, 10.97978687286377, 12.50239086151123, -32.85208511352539, -16.043420791625977, 29.220443725585938, -12.53149127960205, 19.5697021484375, -10.135692596435547, 6.217007637023926, -38.42893981933594, -56.912742614746094, -6.481307506561279, -44.749412536621094, 15.937980651855469, 0.22343246638774872, -9.485658645629883, -25.6039981842041, 25.869890213012695, -9.60351848602295, -39.406761169433594, -34.79142379760742, -5.585357189178467, 42.83032989501953, 5.146176815032959, 33.351985931396484, 11.588007926940918, -15.609481811523438, 6.99464225769043, -9.506708145141602, -37.03738784790039, -11.83639907836914, 2.3161802291870117, -53.41975784301758, 6.094940662384033, 9.933405876159668, -8.870919227600098, -13.337825775146484, -38.12928771972656, 18.221576690673828, -60.908973693847656, -27.941438674926758, 9.715352058410645, 15.784594535827637, -54.003211975097656, 10.686088562011719, 12.563239097595215, 71.85648345947266, 51.688297271728516, -18.477540969848633, -50.78230285644531, -18.466289520263672, 51.350345611572266, -16.312442779541016, -14.330476760864258, -13.793305397033691, 41.949527740478516, -22.65495491027832, 31.00835418701172, 25.73409652709961, -0.6103436350822449, 24.131752014160156, -26.207109451293945, -2.0512685775756836, -0.189436137676239, -19.283313751220703, -4.323368549346924, 31.281452178955078, -6.795209884643555, -46.37309265136719, -17.653291702270508, 28.009185791015625, -10.211833000183105, -13.370698928833008, 55.20692443847656, -11.024662971496582, -48.198089599609375, 1.600527048110962, -3.253049850463867, 6.994174957275391, -15.513876914978027, 17.598787307739258, 49.25830841064453, 32.041290283203125, 75.57838439941406, 15.70927619934082, 8.09913158416748, 15.014607429504395, 22.681644439697266, 31.19449806213379, -8.977212905883789, -22.293119430541992, 22.054725646972656, -44.17899703979492, 9.3616304397583, 24.438066482543945, 1.6764330863952637, 11.400005340576172, 67.93048858642578, 10.798689842224121, -47.20914077758789, 14.576619148254395, -31.72786521911621, -18.31535530090332, 46.41608428955078, -68.35664367675781, 13.974588394165039, 10.715777397155762, -10.496366500854492, 33.95827865600586, -22.870803833007812, -9.754257202148438, -43.92286682128906, -17.902372360229492, -2.819622039794922, -5.343661308288574, -51.084842681884766, 63.998966217041016, -21.94377326965332, -6.414697647094727, 42.9926643371582, -22.123821258544922, -13.644213676452637, -3.6205391883850098, -3.307650327682495, 39.3018798828125, -12.109292030334473, 0.37885531783103943, 23.640117645263672, 19.551849365234375, -68.24813079833984, 4.868902683258057, -15.680249214172363, -8.290143013000488, -8.122465133666992, 9.581942558288574, 6.603023052215576, 21.6018009185791, 9.865828514099121, -15.971742630004883, 24.016199111938477, 8.291620254516602, -32.98758316040039, 10.499286651611328, -23.71150016784668, -59.726661682128906, 2.862246036529541, -4.146762371063232, -3.219494104385376, 4.186450004577637, 18.973716735839844, -14.184599876403809, -4.076552391052246, 46.780616760253906, -53.578609466552734, 27.645803451538086, 16.52817726135254, 15.294914245605469, 23.690900802612305, -42.999603271484375, 33.571895599365234, 26.070636749267578, 36.376930236816406, -0.12390626966953278, -52.5831413269043, -14.180259704589844, 48.24600601196289, 19.206378936767578, 26.434904098510742, -0.695079505443573, 25.0675048828125, 23.62001609802246, -23.96746063232422, 9.414464950561523, 36.372623443603516, 19.86254119873047, -1.91827392578125, -8.244378089904785, -6.702877998352051, -32.71847915649414, -12.850247383117676, 13.040983200073242, 8.922662734985352, -22.396072387695312, -9.811491012573242, -52.221778869628906, -17.253305435180664, -22.252195358276367, 42.01499938964844, -4.753722667694092, -36.44721221923828, -3.0636491775512695, -33.685935974121094, 14.228282928466797, -26.58297348022461, 32.55409240722656, -34.52321243286133, 58.40622329711914, 37.23433303833008, -3.4671897888183594, 16.0297908782959, 25.03807830810547, -36.015968322753906, -41.357601165771484, -19.122817993164062, 5.42535924911499, 30.82841682434082, -16.32662582397461, -49.992218017578125, 13.589994430541992, -18.413902282714844, -13.625452041625977, -18.379825592041016, 22.05807113647461, 27.505855560302734, -19.10942840576172, -8.559284210205078, 30.923316955566406, 19.2032470703125, -22.26323699951172, 6.007014751434326, -20.074825286865234, -26.36815643310547, -21.90036964416504, 20.706106185913086, -16.567310333251953, -4.941110610961914, -9.909372329711914, 50.95277404785156, 13.473494529724121, 30.67723274230957, 22.152273178100586, -52.13251876831055, -51.00174331665039, -11.496484756469727, 32.109405517578125, -24.871355056762695, 58.67562484741211, -25.262737274169922, 11.473916053771973, 29.869726181030273, 19.481595993041992, 78.47525787353516, -30.85746955871582, -23.533870697021484, -32.013389587402344, -2.8902018070220947, 8.597674369812012, -33.74409866333008, -4.39074182510376, -36.150169372558594, -5.82060432434082, -38.755088806152344, -6.789249420166016, 21.605958938598633, 80.18074035644531, 40.529083251953125, 57.107486724853516, -16.760841369628906, 9.208178520202637, -40.51385498046875, 3.535443067550659, 5.616049766540527, -1.4302678108215332, -7.696471691131592, -7.351133823394775, -42.15802764892578, 24.965044021606445, -45.742183685302734, 38.1754264831543, 70.8826675415039, 20.196924209594727, 36.87677001953125, 32.78297805786133, 3.4692282676696777, -24.170631408691406, 41.41846466064453, 18.5175838470459, -40.4638557434082, 69.7300796508789, 0.01663931831717491, -5.713945388793945, 42.135963439941406, -32.942474365234375, 10.163498878479004, -8.063281059265137, 31.06330680847168, 28.80362892150879, -0.8255655169487, 19.116130828857422, -22.5950984954834, 48.42938995361328, -13.570195198059082, 66.58827209472656, 42.29488754272461, -15.369147300720215, 4.024505138397217, -12.966044425964355, -15.016159057617188, -39.376888275146484, 33.67003631591797, 1.0283926725387573, 37.80649185180664, -40.779388427734375, 10.552045822143555, 9.93452262878418, -5.26030158996582, -10.020395278930664, -39.186763763427734, -34.05244827270508, 2.0152812004089355, 21.747642517089844, -32.31443405151367, 28.508371353149414, 41.59779739379883, 54.53676986694336, 1.4065293073654175, 11.452662467956543, -5.077432155609131, 15.332477569580078, 10.255279541015625, 28.71218490600586, -1.37885320186615, 70.50503540039062, 82.1369400024414, 14.520411491394043, 12.476866722106934, -82.14661407470703, -21.029476165771484, 9.857815742492676, -14.077468872070312, -38.81998062133789, -18.161611557006836, -45.992218017578125, -25.845542907714844, -40.71589279174805, 24.298093795776367, -19.965763092041016, 14.879415512084961, -25.76133155822754, -80.20403289794922, 32.6058464050293, -12.71086311340332, -52.82391357421875, 17.787139892578125, 8.494836807250977, 48.301307678222656, -61.31270217895508, 19.619482040405273, 20.100156784057617, 51.5055046081543, 5.6638383865356445, 0.19246064126491547, 13.444994926452637, 29.91303253173828, -23.333471298217773, -10.921335220336914, 2.0351850986480713, 63.52491760253906, 21.716054916381836, -12.777200698852539, -10.647679328918457, 22.211530685424805, -6.379229545593262, 0.49012160301208496, 37.98086929321289, 5.745419025421143, -35.06099319458008, -6.97748327255249, -21.7506046295166, -62.84101867675781, 25.42417335510254, -5.362393379211426, -36.82371520996094, -7.699031352996826, 41.7575569152832, 19.461978912353516, -50.78693389892578, -35.49934768676758, -13.629849433898926, -18.073606491088867, -21.135526657104492, 10.41190242767334, -41.07212829589844, 15.787071228027344, -9.54784870147705, -79.4365463256836, -22.112642288208008, 34.177364349365234, -18.631071090698242, 12.619636535644531, -22.332883834838867, -19.26740837097168, 40.51860809326172, -12.147001266479492, 42.20854949951172, -40.94325637817383, -38.506935119628906, 22.507301330566406, -18.51095962524414, -57.222991943359375, -9.395247459411621, 16.796266555786133, -32.48430252075195, -13.52448558807373, -23.608230590820312, 3.082467555999756, 26.364356994628906, 40.00925064086914, 14.30251693725586, 52.51083755493164, 31.01945686340332, 25.556034088134766, 85.3615493774414, 15.338781356811523, -60.580772399902344, 6.744777202606201, 15.718881607055664, 47.2114143371582, -7.777702331542969, -40.65848922729492, 45.771358489990234, -53.46717834472656, -20.459856033325195, 0.05761365219950676, -8.317201614379883, -18.6245174407959, -3.3439438343048096, -6.759869575500488, -1.0077462196350098, -1.8797712326049805, 10.709633827209473, -46.11939239501953, 19.066673278808594, -14.924052238464355, 48.59032440185547, 15.586162567138672, 13.33720874786377, 28.95611572265625, 45.69834899902344, -25.492233276367188, 60.45146179199219, -27.42735481262207, -5.558774471282959, -78.18203735351562, -3.9975805282592773, -18.784717559814453, -37.56340026855469, 16.196979522705078, -3.8419625759124756, 26.21980094909668, 1.2449475526809692, -9.961994171142578, -53.99079895019531, -15.71277904510498, 50.61272048950195, 1.664811372756958, 15.032395362854004, -23.8337459564209, -26.8836669921875, -82.71996307373047, -4.458261966705322, 31.773536682128906, -4.157978057861328, 14.033350944519043, 47.134178161621094, 24.470346450805664, 16.655855178833008, -12.846375465393066, -41.101348876953125, 25.030065536499023, 16.372303009033203, 12.237586975097656, 38.097537994384766, 42.59306716918945, -18.144695281982422, 56.875736236572266, 17.309310913085938, 47.413204193115234, 14.091962814331055, -39.62256622314453, -13.042486190795898, -27.73613739013672, -29.926715850830078, -12.316667556762695, -6.192807674407959, -39.27284240722656, 24.074748992919922, -37.6947021484375, 3.3783650398254395, 23.3809757232666, -29.50597381591797, 5.423673629760742, -6.8106889724731445, 10.857256889343262, -19.592041015625, -14.982787132263184, -2.922257900238037, -64.8105239868164, 43.572410583496094, -1.5149741172790527, -15.563899040222168, -11.68498706817627, -1.8463258743286133, -12.269420623779297, 27.088233947753906, 22.680097579956055, -30.14624786376953, 69.87191772460938, -9.969047546386719, 11.152640342712402, 9.30844497680664, -23.582500457763672, -26.548765182495117, 14.306946754455566, -16.537954330444336, -29.243972778320312, 11.89572525024414, -11.030706405639648, -25.632158279418945, -41.84513473510742, 43.194984436035156, 4.455759525299072, 6.581557750701904, -21.453813552856445, -35.453651428222656, -20.797077178955078, -29.666980743408203, 45.5925178527832, 37.45526123046875, -30.358356475830078, 9.329480171203613, 2.1916704177856445, -8.940937995910645, 18.16639518737793, 52.98395538330078, -42.261619567871094, 55.14010238647461, -29.469511032104492, 24.312850952148438, -62.83904266357422, -39.160003662109375, -23.514698028564453, -11.561392784118652, -31.19452667236328, 41.3428955078125, -1.4709392786026, -1.9531240463256836, -21.151838302612305, -15.240687370300293, 51.52887725830078, 16.21657371520996, -40.00175476074219, -16.180809020996094, -1.9628257751464844, 1.304076910018921, -76.38046264648438, 17.22808265686035, -4.115843296051025, 13.67296314239502, 27.471811294555664, -62.00627136230469, -3.134394645690918, 8.581027030944824, 22.389602661132812, -15.014554977416992, 40.386226654052734, -26.853784561157227, -23.551467895507812, 9.138964653015137, -5.907044410705566, 17.08781623840332, -10.932132720947266, 27.710453033447266, -18.90529441833496, 62.12716293334961, 2.3001010417938232, 15.137320518493652, 29.46681785583496, 30.679853439331055, -27.606529235839844, 35.13754653930664, -39.41477966308594, -51.375328063964844, 8.55378532409668, 38.91437530517578, 17.30119514465332, 14.352575302124023, 69.53408813476562, 28.463014602661133, 31.612680435180664, 0.15348610281944275, 23.551898956298828, -24.406091690063477, -5.552158832550049, -19.65009307861328, 65.09805297851562, -40.83476257324219, -10.630908966064453, -19.563753128051758, -1.940049648284912, 27.2048282623291, -18.633604049682617, -16.5802059173584, -42.41047668457031, -10.615999221801758, -26.387420654296875, -21.77136993408203, 38.311988830566406, 32.85541534423828, -25.732696533203125, 2.978332996368408, -49.61554718017578, 17.107677459716797, -19.247343063354492, 0.3219810724258423, -6.102657794952393, 46.435157775878906, 5.37098503112793, 32.5560302734375, 2.5703258514404297, 37.773475646972656, -2.2263317108154297, -14.615701675415039], [12.233482360839844, -26.693872451782227, -26.209888458251953, 13.099227905273438, -34.87959289550781, 5.831538677215576, 10.695538520812988, -68.09600067138672, -1.9614131450653076, 40.930763244628906, -31.189266204833984, -4.729029655456543, 59.39487075805664, -18.2983455657959, 14.694107055664062, -27.013381958007812, 41.211830139160156, -45.22000503540039, -69.33100891113281, 21.433956146240234, -21.469751358032227, -59.203834533691406, 37.88430404663086, 11.261456489562988, -21.514612197875977, -16.194469451904297, 21.2628173828125, -33.45790100097656, -21.40279769897461, 11.08289909362793, -15.439477920532227, 4.1826171875, -35.652103424072266, -9.0927734375, 13.102962493896484, 15.038108825683594, 2.627159357070923, 40.0363655090332, 6.28521728515625, -33.272979736328125, 31.7619571685791, 27.391311645507812, -7.402797698974609, -16.6020565032959, 11.716024398803711, 23.434030532836914, -67.84275817871094, -7.363768577575684, -33.716392517089844, -43.76243209838867, 13.276206970214844, 15.56911849975586, -2.5796256065368652, -3.3595693111419678, 52.962074279785156, 16.87088394165039, 10.213713645935059, -14.781349182128906, -14.562941551208496, -14.081005096435547, -0.21482904255390167, 24.696142196655273, -5.873562335968018, -11.575984954833984, -22.345020294189453, -53.964839935302734, 16.131175994873047, -21.771263122558594, -59.67266845703125, -9.901320457458496, 66.32052612304688, -9.099424362182617, 29.99539566040039, 2.4967727661132812, -15.594099998474121, 18.03563117980957, -9.241701126098633, 17.501096725463867, 3.9160985946655273, -25.393001556396484, -8.05675220489502, -17.929107666015625, 31.985288619995117, -56.10203170776367, -20.595169067382812, 6.519023895263672, -29.961788177490234, -26.727863311767578, -37.495140075683594, 5.996530055999756, 0.2924821972846985, -71.83820343017578, 1.7695618867874146, -34.26585388183594, -16.440866470336914, -10.289076805114746, -20.506084442138672, -33.21815490722656, -38.00697326660156, 7.553477764129639, 36.50069808959961, 16.211772918701172, -9.379012107849121, 4.534671783447266, -49.170135498046875, -16.194459915161133, 10.390034675598145, -27.978179931640625, 61.014427185058594, 7.806380748748779, 10.862314224243164, -41.263221740722656, 25.801448822021484, 30.54449462890625, -29.035558700561523, 25.122753143310547, -44.988895416259766, 40.902244567871094, -15.754729270935059, 13.419960021972656, 75.77748107910156, -8.999906539916992, 41.718101501464844, 25.370880126953125, 24.433385848999023, -25.28221893310547, 24.762388229370117, -42.8213996887207, 31.366641998291016, 28.609498977661133, 69.29910278320312, 15.337077140808105, -15.524682998657227, 29.586742401123047, -38.730628967285156, -14.531648635864258, 51.25644302368164, 15.30863094329834, -11.469884872436523, -43.34210968017578, 12.698476791381836, 13.831925392150879, 17.86635398864746, 7.02992057800293, 40.41246795654297, 41.019187927246094, -9.21848201751709, -13.043327331542969, -29.850540161132812, -3.346501350402832, 4.111420631408691, 2.2479939460754395, 60.41365432739258, 26.043357849121094, 7.380670070648193, -28.40315818786621, -36.828033447265625, -28.853824615478516, -28.71408462524414, -2.0267109870910645, 27.806421279907227, 7.499475955963135, 1.3198580741882324, -22.012117385864258, -2.5237948894500732, 13.602237701416016, 13.812687873840332, 64.3246841430664, -2.0197267532348633, 25.995704650878906, -18.511194229125977, -8.507869720458984, 18.92148208618164, 33.436771392822266, -5.572999954223633, 12.008072853088379, 21.850418090820312, 7.445219039916992, -21.05177116394043, 5.2137370109558105, -25.546546936035156, -17.07657241821289, 0.29984307289123535, 13.08104419708252, -9.091160774230957, -78.44622039794922, -32.7010612487793, -39.70103073120117, 3.9565913677215576, -1.1934140920639038, 16.44589614868164, -29.605268478393555, 33.66227340698242, 28.89756202697754, -32.18095779418945, 23.377099990844727, -18.30495834350586, 25.013736724853516, -32.40176773071289, 20.981842041015625, -30.018404006958008, 37.75957489013672, 6.6523942947387695, -15.118703842163086, -5.4597930908203125, -15.501859664916992, 8.65700626373291, -4.40978479385376, -11.109551429748535, -10.11395263671875, -51.358253479003906, -56.92654037475586, -0.6045536398887634, -35.474815368652344, -42.14056396484375, 5.358010768890381, 32.09429168701172, 42.555973052978516, 7.85639762878418, -33.45766830444336, 42.02068328857422, -37.3657341003418, -33.381099700927734, 6.724051475524902, 5.143515586853027, -6.5728607177734375, 10.77756118774414, 21.715219497680664, -21.00273895263672, -19.107938766479492, 70.04481506347656, 32.9833984375, -1.0413092374801636, -59.26325225830078, -30.434667587280273, -7.385062217712402, -1.1034456491470337, 28.36374855041504, -11.44325065612793, -0.32538142800331116, 7.206136226654053, -24.884735107421875, 0.6692432761192322, -30.41294288635254, 16.0295352935791, -43.089378356933594, -49.707740783691406, -44.021453857421875, 0.8881019353866577, 4.314088821411133, 26.678176879882812, -11.324590682983398, 47.014732360839844, 7.569109916687012, -21.84287452697754, -33.607425689697266, 50.737369537353516, 55.723018646240234, -7.878385543823242, -35.22200012207031, 19.159223556518555, -25.027915954589844, -34.849098205566406, -63.7882080078125, -5.585358142852783, -37.94087219238281, 3.5512096881866455, -47.6372184753418, 19.821697235107422, -28.86688232421875, 29.860504150390625, 30.89988899230957, 18.198551177978516, 17.578210830688477, -15.93620491027832, -7.617389678955078, -72.0511474609375, -11.88952922821045, 3.4840712547302246, 63.87608337402344, -2.233879804611206, 1.592018723487854, 23.830249786376953, -31.374574661254883, -9.118315696716309, -8.358746528625488, -77.59614562988281, -20.622295379638672, -10.409872055053711, 34.10618209838867, 7.0689520835876465, 32.97269058227539, 21.738384246826172, 25.318376541137695, -64.84447479248047, 86.92724609375, -6.730100154876709, 1.9245551824569702, 28.203083038330078, -27.625673294067383, 6.603521823883057, 14.718049049377441, -19.569019317626953, -5.325061321258545, -21.13753890991211, -13.693634986877441, 36.453765869140625, -53.13773727416992, 21.829753875732422, -26.596805572509766, 54.35002517700195, 32.433536529541016, 17.271257400512695, -9.85770034790039, -27.456130981445312, -75.40324401855469, -14.341822624206543, -15.661962509155273, -2.886861801147461, 3.1256766319274902, 15.96438980102539, -2.6763689517974854, 4.314669132232666, -25.325294494628906, -59.14988327026367, -43.49894332885742, -42.73735809326172, 25.805721282958984, -35.86977767944336, 45.72844696044922, -7.72188138961792, -18.88660430908203, 6.433784008026123, -29.570829391479492, -18.726913452148438, -14.47401237487793, -8.395508766174316, 20.111181259155273, 40.78181076049805, 4.845572471618652, 42.71882629394531, -2.397676706314087, 28.45711326599121, 12.948861122131348, -32.83589172363281, 20.597091674804688, 8.402840614318848, -17.615581512451172, 4.433969020843506, -14.584693908691406, -2.708317279815674, -4.556427955627441, 8.128093719482422, -10.162546157836914, 8.907602310180664, -52.4453010559082, 14.145356178283691, 30.018232345581055, -22.65390396118164, 16.641517639160156, -31.028444290161133, -25.946147918701172, 12.181694030761719, 16.46085548400879, -10.194190979003906, 8.932558059692383, 4.7389960289001465, 21.481821060180664, 14.786928176879883, 21.11395835876465, 41.86311340332031, 28.64957046508789, 16.956445693969727, -48.31001663208008, 7.283519268035889, -38.16448974609375, 17.163921356201172, -49.571659088134766, 38.88545608520508, -31.858909606933594, 41.767173767089844, -13.393041610717773, 4.6104631423950195, 39.78542709350586, 7.081242561340332, 12.120223045349121, -0.9722778797149658, -35.84059524536133, -56.599788665771484, -23.617450714111328, 23.753692626953125, 16.8510799407959, 71.52046966552734, 46.42649459838867, 34.36189651489258, -48.5135498046875, 38.75413131713867, -53.98066711425781, -72.18717193603516, 49.88400650024414, -14.34489631652832, 41.79344940185547, -24.000545501708984, 49.78904724121094, 3.4226455688476562, 20.863033294677734, -54.45200729370117, -15.774482727050781, 11.088971138000488, 5.102062702178955, -17.998483657836914, 39.89698791503906, -24.246477127075195, 15.390028953552246, 14.955337524414062, 14.516632080078125, 20.366687774658203, 39.05513000488281, 2.318680763244629, -30.655881881713867, 1.512640118598938, 19.391098022460938, 27.35019302368164, 9.5005521774292, 27.605785369873047, -9.6952486038208, -16.965404510498047, 10.13150691986084, -30.284921646118164, 19.61701202392578, -49.8293342590332, 3.541022777557373, -25.103878021240234, -31.087596893310547, -62.93174743652344, -16.720890045166016, -43.106815338134766, -8.769695281982422, 14.239453315734863, -7.566575050354004, 33.698272705078125, 38.81967544555664, 8.712320327758789, 59.51072311401367, -0.8030263781547546, 6.385648250579834, -31.957725524902344, -21.71879768371582, 12.875118255615234, -17.51626968383789, -25.74407196044922, 15.919142723083496, -16.800975799560547, -23.815107345581055, -28.986677169799805, -23.737215042114258, 17.456302642822266, 18.854143142700195, 19.73744010925293, 9.33934211730957, 10.893777847290039, 3.1186277866363525, -12.937568664550781, 62.42507553100586, 2.753352403640747, 21.006288528442383, -35.13456726074219, -11.56994342803955, 23.63227653503418, 48.316650390625, 21.271211624145508, 18.85286521911621, 23.700103759765625, -28.166587829589844, 4.4774627685546875, 22.072677612304688, 9.630148887634277, -7.315349578857422, 6.04433536529541, 71.77613067626953, -34.491119384765625, 26.08191680908203, 32.45087432861328, -5.952716827392578, 17.61301612854004, 35.34962463378906, 32.74687576293945, 45.48605728149414, -54.22095489501953, 14.174864768981934, -13.61217212677002, -30.486778259277344, 34.244754791259766, -24.2194766998291, 0.09190592914819717, 49.842681884765625, -35.322906494140625, 6.959643363952637, 88.69623565673828, 10.89090347290039, -0.8368780016899109, 52.127410888671875, 39.819427490234375, -7.998044967651367, -43.47098922729492, -9.133110046386719, 3.2571797370910645, -26.972898483276367, -1.7590911388397217, -21.980209350585938, 20.044769287109375, -26.100847244262695, -15.918071746826172, 2.845411539077759, -43.200260162353516, -32.33653259277344, -49.20487976074219, -14.75579833984375, 24.06598663330078, -8.550765037536621, 9.813961029052734, 18.93206214904785, 59.31748580932617, 15.439628601074219, 33.878211975097656, -18.581892013549805, 33.56930923461914, 29.757532119750977, 19.918859481811523, -29.168882369995117, -18.725997924804688, 55.737342834472656, -19.231727600097656, -1.2563800811767578, 8.174396514892578, 49.343345642089844, 9.665328979492188, 10.92756462097168, 38.305084228515625, -26.304141998291016, 5.899506568908691, -2.6869211196899414, 17.583616256713867, 49.1630859375, 62.122650146484375, -39.13821029663086, 22.695676803588867, -2.2911980152130127, -24.21129035949707, -21.99570083618164, -62.746517181396484, 45.380977630615234, -25.925594329833984, -37.66819763183594, 17.690208435058594, 32.648712158203125, -7.707230091094971, -22.692392349243164, -7.4861626625061035, -65.71353912353516, 21.70700454711914, -19.205547332763672, 12.121574401855469, 23.31835174560547, 10.746295928955078, 9.5033597946167, -1.2167153358459473, -69.80500793457031, 26.51633071899414, 53.30746078491211, 4.943373680114746, -21.081926345825195, 7.140960693359375, -29.050090789794922, 46.02668762207031, 35.91239929199219, 34.91249084472656, 14.825861930847168, 43.62488555908203, -5.225465774536133, 41.61155700683594, -18.816062927246094, -18.871864318847656, 2.266874074935913, 22.987533569335938, -10.603496551513672, 35.58361053466797, -6.386607646942139, -24.370563507080078, 65.80229187011719, -22.362525939941406, 2.73667049407959, -30.400100708007812, 34.14430236816406, 8.63264274597168, 21.134910583496094, -23.832605361938477, -41.28436279296875, -15.963851928710938, -4.449525356292725, -24.203838348388672, 13.525928497314453, -29.31869888305664, -18.417762756347656, -8.619412422180176, 34.77328109741211, 5.306045055389404, 19.407649993896484, 4.814620494842529, 0.9813067317008972, -12.445880889892578, -25.815799713134766, 23.950855255126953, -22.780391693115234, -33.678409576416016, 14.320867538452148, -30.89910125732422, 16.070362091064453, 5.939389705657959, 12.943037986755371, 23.68747901916504, -12.82406997680664, 14.141439437866211, 3.361412525177002, 20.59037971496582, -25.21591567993164, 15.519525527954102, 40.850608825683594, -13.577103614807129, 0.3582843542098999, -27.78089714050293, -21.86676025390625, -21.448566436767578, -6.307211875915527, -39.02488327026367, 43.29218673706055, -0.8543097972869873, 44.37281799316406, -13.424431800842285, -34.500003814697266, -36.25545883178711, -25.025039672851562, 17.67152976989746, 20.066020965576172, -30.047653198242188, 16.917831420898438, -29.19340705871582, 24.462926864624023, 32.69799041748047, 36.76633834838867, -34.91733932495117, 10.60403060913086, 21.152563095092773, 34.58659744262695, -60.8575553894043, -2.0926029682159424, -6.828228950500488, -18.752056121826172, 33.23001480102539, -22.68913459777832, 30.996482849121094, -25.620519638061523, -44.62348937988281, 21.075973510742188, 18.22808837890625, -11.855412483215332, -18.185569763183594, -27.9201602935791, 33.19427490234375, -57.936912536621094, -55.002254486083984, -23.91629409790039, -17.454988479614258, 32.907989501953125, -5.902400016784668, -5.960602283477783, -38.564762115478516, 7.510935306549072, -17.15317726135254, -0.17957091331481934, 22.162715911865234, -7.94985818862915, -18.862659454345703, 12.245576858520508, -20.008975982666016, 38.97610092163086, 12.156754493713379, -0.3069165349006653, -10.445304870605469, 25.870662689208984, -7.980316638946533, -7.3996171951293945, -6.478110313415527, -20.367380142211914, -51.775146484375, -11.998336791992188, -59.24580383300781, 18.920738220214844, 79.08641052246094, -4.038661479949951, -1.5067470073699951, -17.88836669921875, 52.10519790649414, -26.341266632080078, 4.497512340545654, -12.117697715759277, 8.957376480102539, 13.756956100463867, 24.656856536865234, -33.70741271972656, 0.4074820876121521, -22.966190338134766, 28.714094161987305, -19.538711547851562, 11.779036521911621, -13.186528205871582, 28.25922393798828, -21.534276962280273, -2.0979995727539062, -50.11253356933594, -14.164192199707031, 37.54804992675781, -34.78974151611328, 40.208213806152344, -8.816431999206543, -23.340749740600586, -6.772968769073486, -17.66270637512207, 23.942203521728516, 36.07948303222656, -6.98394250869751, -24.39512062072754, -35.79048156738281, 9.59503173828125, -24.76224136352539, 17.551166534423828, 10.328788757324219, 14.455410957336426, -7.892629623413086, 17.295921325683594, 29.198440551757812, 32.98996353149414, -0.031946029514074326, -6.810798645019531, -16.635761260986328, 17.44573211669922, 17.798099517822266, -16.72708511352539, 59.697566986083984, 9.196548461914062, 32.10847473144531, -33.6768684387207, -14.19443130493164, 33.33866500854492, -21.524341583251953, -22.03701400756836, -14.688263893127441, 6.440151691436768, -14.224382400512695, -3.8823885917663574, 44.6469612121582, -22.238786697387695, 9.977736473083496, 11.000741958618164, -35.11521530151367, -25.91209602355957, 23.390283584594727, 27.52179718017578, 44.0760383605957, 1.4842782020568848, -7.360799312591553, -22.40749168395996, -2.102231502532959, 10.196332931518555, 18.963308334350586, 1.6515662670135498, -6.654108047485352, 34.07007598876953, -14.231566429138184, 2.89127516746521, 20.891447067260742, -29.33584213256836, 16.562807083129883, -59.16059875488281, 2.878277063369751, -28.14784049987793, -18.600799560546875, -4.939012050628662, 26.07550048828125, 60.007015228271484, -13.435282707214355, -34.932891845703125, 41.72401809692383, 11.960881233215332, -8.904464721679688, 5.308665752410889, 7.515415668487549, 12.317702293395996, -19.79871940612793, 45.751617431640625, 38.80027389526367, 49.28740310668945, -8.506556510925293, -12.976211547851562, 29.288002014160156, 88.10462951660156, 10.033157348632812, 17.68726348876953, 33.5295524597168, -4.202243328094482, 31.216707229614258, -2.1529321670532227, 22.690752029418945, -56.69428634643555, 27.015657424926758, -7.57072639465332, -29.121015548706055, 39.172828674316406, -44.33357620239258, 11.843263626098633, 22.702722549438477, -3.4970314502716064, 100.0773696899414, -7.60572624206543, 28.733264923095703, -3.6692428588867188, -16.447355270385742, 18.106491088867188, 3.8396828174591064, -25.635948181152344, 26.19931411743164, -19.467159271240234, -28.182140350341797, -15.153910636901855, 35.16372299194336, -32.92835235595703, -1.4996235370635986, 2.8324711322784424, -70.6565170288086, -2.6851015090942383, -8.922541618347168, -39.31229782104492, -30.458152770996094, 7.21188497543335, -14.649401664733887, -39.59706497192383, 38.14159393310547, 28.236268997192383, -12.61472225189209, 29.445276260375977, 3.543341636657715, 23.249095916748047, -61.78020095825195, -26.56163787841797, -1.656059741973877, 53.613521575927734, 2.771806001663208, 17.609535217285156, -30.652301788330078, -22.496780395507812, 32.60860824584961, 25.543867111206055, 12.798196792602539, 0.7823070883750916, -20.554927825927734, 5.33943510055542, -16.46472930908203, -39.588623046875, 34.1153678894043, 4.684838771820068, 54.21192169189453, 13.968323707580566, 2.5604934692382812, 41.16229248046875, -18.104101181030273, 2.235694646835327, -1.4609743356704712, -15.292593002319336, -11.27288818359375, 30.059406280517578, 9.271194458007812, 30.777490615844727, -17.771936416625977, 14.169816017150879, 17.43027114868164, -10.055015563964844, 58.38047790527344, 0.8328936696052551, -26.71198081970215, -32.34729766845703, -53.218849182128906, -8.772637367248535, -16.402769088745117, 40.87321090698242, 50.163875579833984, -1.4034125804901123, 12.703837394714355, 18.366750717163086, 21.824100494384766, -5.985745429992676, 28.141864776611328, -43.927772521972656, -65.6710205078125, 14.751795768737793, -6.963533878326416, 45.77763748168945, 1.7808347940444946, 23.844884872436523, 9.960673332214355, 20.73919105529785, 13.168366432189941, -7.237011432647705, -5.901189804077148, 30.669275283813477, 19.852571487426758, -45.841434478759766, -56.429073333740234, -37.455474853515625, -6.83412504196167, 75.03038787841797, -23.849327087402344, 25.581811904907227, -53.887447357177734, -9.897832870483398, 57.28083801269531, -1.0294989347457886, 24.200742721557617, -42.32965087890625, 25.16707992553711, 14.97531795501709, -38.89064025878906, 3.4664833545684814, -13.16367244720459, 44.65090560913086, -12.768726348876953, 33.86208724975586, 28.479345321655273, -46.509735107421875, 20.462158203125, -3.2285525798797607, 2.4951515197753906, -3.4323463439941406, -40.01089859008789, -22.744421005249023, 21.900772094726562, 53.95931625366211, 26.29312515258789, 3.1622371673583984, 13.498745918273926, -15.30090618133545, 16.330921173095703, -26.120697021484375, -40.04826354980469, 41.49836730957031, -13.836836814880371, -37.59199905395508, -14.759072303771973, 3.3637712001800537, 4.0976080894470215, 32.99795150756836, 23.114133834838867, -12.57139778137207, -37.76480484008789, 18.360891342163086, -56.29057693481445, 17.09288787841797, -3.683941602706909, -12.94234561920166, -45.083351135253906, -27.124725341796875, 44.181941986083984, -14.072030067443848, 7.210893154144287, 6.785682201385498, -19.586511611938477, -8.782718658447266, 36.54593276977539, -48.124549865722656, 22.227378845214844, 35.95925521850586, -50.03476333618164, -67.74771881103516, 4.774725437164307, -0.5239080786705017, 13.64321231842041, 4.263982772827148, -47.42694854736328, 10.077018737792969, 18.26500129699707, -9.415977478027344, -0.05247020348906517, 15.313521385192871, 8.12995433807373, 51.2352409362793, -10.944329261779785, 35.20963668823242, -45.83413314819336, 54.77986145019531, 51.87238693237305, -22.84183120727539, -31.73645782470703, -4.851349830627441, 0.4257180690765381, 1.7523717880249023, -44.499332427978516, 4.532933712005615, -17.218082427978516, -54.76848602294922, -11.912732124328613, 45.12603759765625, 17.44970703125, -35.65535354614258, -36.67275619506836, -29.8823299407959, -15.688220977783203, -8.940339088439941, -17.698970794677734, -26.08537483215332, 11.140636444091797, 2.550083637237549, 17.68506622314453, 35.99766540527344, 0.6006673574447632, 10.755308151245117, -15.291616439819336, -9.691726684570312, 10.556838035583496, -4.30104923248291, 33.85850143432617, -10.2283935546875, 3.263380765914917, -22.2641658782959, -10.497090339660645, -54.09947967529297, -23.273313522338867, -38.37287902832031, -6.938451766967773, -27.441631317138672, 4.105534553527832, -0.49352312088012695, 0.3286016285419464, -33.21833801269531, -35.96987533569336, 38.94921875, -7.903242588043213, 25.935325622558594, -10.862759590148926, 75.92782592773438, -1.5724579095840454, 19.525163650512695, -35.574974060058594, -8.041963577270508, 26.668102264404297, 13.744357109069824, 48.47137451171875, 33.665748596191406, -4.4996747970581055, 5.0550923347473145, -42.49723434448242, 1.9588268995285034, -33.81132125854492, -17.425434112548828, -28.246417999267578, 95.32221984863281, -38.10340118408203, -19.402904510498047, 127.73519134521484, 18.638742446899414, -33.3702392578125, -9.242061614990234, -12.428764343261719, 33.80004119873047, -2.8090054988861084, 11.530577659606934, 28.697418212890625, 62.65723419189453, 5.580074787139893, 30.63983917236328, -6.324343204498291, -18.27775764465332, -29.2632999420166, -2.8260233402252197, 22.259876251220703, 16.410259246826172, 23.854291915893555, -48.40707778930664, -68.6300277709961, 46.157291412353516, 48.21736526489258, 27.470006942749023, -11.704734802246094, 10.799013137817383, 35.70415496826172, 47.5538330078125, 27.841148376464844, 2.36513614654541, 19.480371475219727, 1.2469735145568848, -20.588716506958008, -27.107585906982422, 1.261508822441101, 23.503786087036133, -9.499334335327148, 8.500770568847656, -17.999706268310547, -1.9276008605957031, 20.55536460876465, 9.721596717834473, 43.55951690673828, 2.3579256534576416, 22.537382125854492, -13.85910415649414, -14.931930541992188, 5.560919761657715, -16.503190994262695, 60.915321350097656, 25.81929588317871, -11.945049285888672, -15.084634780883789, 14.110191345214844, 12.540861129760742, -14.742783546447754, -83.43230438232422, -11.566635131835938, -29.271408081054688, -14.476515769958496, 20.544097900390625, 2.4554028511047363, 24.174598693847656, -11.099567413330078, -27.642393112182617, 15.438430786132812, -38.827945709228516, 0.4183327555656433, 11.142404556274414, 18.319765090942383, -35.56416702270508, -1.8559954166412354, -6.4592084884643555, -10.389352798461914, 14.71120834350586, 28.827577590942383, -12.586461067199707, -27.965625762939453, 27.732105255126953, -8.109655380249023, -35.239845275878906, 1.8958637714385986, -35.39997863769531, -3.853926658630371, -5.01418924331665, -7.751307010650635, 39.61651611328125, -19.820926666259766, -24.504560470581055, 68.51818084716797, 4.045439720153809, -6.495508670806885, -0.37034058570861816, -30.44768524169922, -2.229898691177368, -21.4593448638916, 13.31241226196289, -12.079720497131348, -34.02464294433594, -32.244140625, -13.851090431213379, 28.695466995239258, 3.2772819995880127, 32.96263122558594, -8.926108360290527, -31.4653263092041, 2.730001449584961, -41.21513366699219, -12.501604080200195, 34.272464752197266, 48.29670715332031, -21.679828643798828, -25.955007553100586, -21.37713050842285, -4.473863124847412, -19.198392868041992, 39.399864196777344, -33.20469284057617, -20.368633270263672, -24.691164016723633, -3.600881338119507, 35.54746627807617, -31.885421752929688, 22.076229095458984, -29.25825309753418, 19.33424949645996, 1.1611456871032715, 1.1041285991668701, 2.405820846557617, 8.391313552856445, -9.624913215637207, 36.21042251586914, -30.4158992767334, 26.32640266418457, 3.701201915740967, 20.92281150817871, 24.211362838745117, -19.661239624023438, 15.401151657104492, 8.677639961242676, 14.02664852142334, 14.23046875, 4.6401448249816895, 61.790992736816406, 55.416473388671875, -1.855621099472046, 31.817472457885742, 36.03742218017578, 11.842272758483887, -38.21350860595703, 57.88274383544922, 32.344093322753906, -0.4252876043319702, -20.881277084350586, -12.679586410522461, -4.594727516174316, 7.216320514678955, 17.232648849487305, 26.245758056640625, 69.496826171875, 8.949129104614258, -16.08488655090332, -13.593427658081055, 18.725522994995117, 37.26138687133789, -39.58522033691406, 71.3827133178711, 28.287328720092773, 22.180866241455078, 16.58742332458496, -15.864666938781738, -10.856330871582031, 2.2324717044830322, 21.08244514465332, 22.00529670715332, 23.043384552001953, -29.61931610107422, -12.247769355773926, -5.591012001037598, 59.755767822265625, -45.34196853637695, 15.938779830932617, -30.094684600830078, 54.99359893798828, -22.0120906829834, -15.58076286315918, -20.22972297668457, 22.09893226623535, 6.783960342407227, -21.550291061401367, -25.56087303161621, -26.202068328857422, 36.5806770324707, 12.7366943359375, 37.88808822631836, -12.42782974243164, -12.0087890625, -8.886880874633789, -11.319644927978516, -37.23460388183594, -12.570967674255371, -58.47315216064453, 8.69448184967041, -4.943028926849365, 22.924888610839844, 52.82126235961914, 35.68912887573242, 50.01799011230469, 26.215776443481445, 18.263437271118164, 10.718111991882324, 48.761207580566406, -27.179162979125977, 7.227477550506592, -37.16482925415039, 36.66672134399414, -15.780047416687012, -13.420502662658691, -13.340968132019043, 0.4481608271598816, -16.08646011352539, 14.774422645568848, 36.622772216796875, -32.94843292236328, 60.232994079589844, 5.6121063232421875, -21.177837371826172, 6.966881275177002, 19.22959327697754, 29.23865509033203, -5.27799129486084, 16.860614776611328, -30.569936752319336, 14.01950454711914, -13.946088790893555, 31.53769302368164, 78.92061614990234, -54.03324890136719, 28.213659286499023, 77.75860595703125, -10.652522087097168, -29.898775100708008, 20.573965072631836, 2.529184341430664, -13.449883460998535, 29.608409881591797, -7.2271928787231445, 32.439666748046875, 12.387862205505371, 5.8887200355529785, -28.87021827697754, -32.40205764770508, -23.179731369018555, 25.200546264648438, -15.99203872680664, 27.566469192504883, 15.98079776763916, 21.061975479125977, 58.890811920166016, -25.067291259765625, 11.636713981628418, -7.245364665985107, 33.97216033935547, -20.616413116455078, -64.2838134765625, -21.657638549804688, 16.645416259765625, 8.901999473571777, 29.603775024414062, 30.603811264038086, 38.80671310424805, 3.947463274002075, -33.51844787597656, -16.156108856201172, 18.35337257385254, -46.59938049316406, 15.5301513671875, -16.13776969909668, 7.337229251861572, 65.65980529785156, -27.563472747802734, -17.303537368774414, 47.70783615112305, -0.8291416764259338, 13.72175121307373, 7.899488925933838, 48.942909240722656, 19.603702545166016, -0.41575363278388977, -36.645599365234375, 13.364251136779785, -22.951326370239258, -69.97946166992188, -69.87297058105469, -44.19713592529297, 93.79609680175781, 26.562509536743164, 13.339061737060547, 6.157899856567383, -7.145602226257324, -20.890045166015625, -75.34283447265625, -8.176483154296875, 20.196365356445312, -19.292200088500977, 22.231449127197266, -1.5398921966552734, -19.370254516601562, -46.595726013183594, -23.29213523864746, -3.8490681648254395, -18.999120712280273, -20.124717712402344, -45.974090576171875, -42.047271728515625, 35.53473663330078, 8.388795852661133, -9.928956031799316, -9.99245548248291, 24.30543327331543, 58.858970642089844, -27.841646194458008, 6.699743747711182, 42.94364547729492, 11.883051872253418, -40.229881286621094, 56.802574157714844, -33.875492095947266, -19.23049545288086, 37.96376419067383, -12.898358345031738, 35.282466888427734, 41.98686599731445, -3.963571786880493, -6.5001630783081055, 46.511234283447266, -1.2809362411499023, -26.338336944580078, -7.340918064117432, 7.203945636749268, -5.907068252563477, 9.042708396911621, 8.699240684509277, -36.247440338134766, -34.517276763916016, 2.815345525741577, 31.077110290527344, -79.20234680175781, -6.37321138381958, 13.884196281433105, 23.775630950927734, 13.528411865234375, -28.946014404296875, 16.951723098754883, -26.92904281616211, -27.9487361907959, -21.417579650878906, 36.00566864013672, 19.21041488647461, 40.149566650390625, 27.32219696044922, 0.29533085227012634, -14.041085243225098, -47.07600784301758, -26.98904800415039, 18.893795013427734, 42.89026641845703, 10.789243698120117, 59.07261657714844, -13.541605949401855, 61.12089157104492, 4.9679059982299805, 29.185056686401367, -16.979476928710938, -12.737810134887695, 3.943809747695923, 30.34232521057129, 27.855117797851562, 8.242586135864258, -38.12777328491211, 23.46238136291504, -8.13565731048584, -20.051895141601562, -59.736976623535156, 49.19921875, 53.60014724731445, -38.795692443847656, 7.7675886154174805, -28.371871948242188, -3.7892916202545166, -28.15919303894043, 27.2099609375, -5.505387783050537, -2.80930757522583, 14.207878112792969, -32.076148986816406, -101.85592651367188, 48.817100524902344, 5.965728282928467, 2.0911121368408203, -0.22996015846729279, -2.6428699493408203, -19.089296340942383, -58.43974304199219, -26.678590774536133, -13.003365516662598, -7.47903299331665, 14.032286643981934, 33.99978256225586, 0.09146830439567566, 18.425323486328125, 40.37892150878906, 59.79509353637695, -17.112741470336914, -2.787187337875366, 21.971933364868164, -46.600486755371094, -22.008901596069336, -10.499692916870117, -36.24842071533203, 32.12690353393555, 16.237157821655273, 78.72303771972656, -6.667771816253662, 31.858814239501953, 8.482566833496094, 44.68315887451172, 48.705345153808594, -16.912090301513672, -26.172815322875977, -0.530763566493988, -13.64579963684082, 14.697883605957031, 10.479191780090332, 15.821313858032227, -45.449462890625, 18.544401168823242, -12.19276237487793, -28.87717628479004, 15.705805778503418, 37.65744400024414, -10.88817024230957, 24.489591598510742, 14.366004943847656, 13.859964370727539, 17.77312660217285, -21.811288833618164, -27.15411949157715, 37.38438415527344, 14.968602180480957, 25.897872924804688, -47.80857849121094, -10.38507080078125, 53.73956298828125, 46.531131744384766, 17.92729949951172, -9.149391174316406, -26.204246520996094, -13.693037033081055, -45.93165588378906, -27.203289031982422, -28.301769256591797, 34.126197814941406, -3.4998316764831543, 18.976037979125977, -14.695169448852539, 29.06882667541504, -38.08831787109375, -22.276960372924805, 27.80772590637207, -21.887859344482422, -6.717517375946045, -1.1758302450180054, 25.180070877075195, -8.641236305236816, -42.5120964050293, 24.20905113220215, 48.482818603515625, -40.68285369873047, 45.271324157714844, 58.38783264160156, -38.44673538208008, 38.658042907714844, 22.133676528930664, -0.6696038842201233, -1.7670880556106567, 10.661436080932617, -23.448514938354492, -3.9054040908813477, -27.77449607849121, 47.22928237915039, 3.75620174407959, -14.246447563171387, 11.801839828491211, 1.5336952209472656, 3.6994612216949463, -26.342506408691406, -1.2323479652404785, 0.245402991771698, 53.408016204833984, -6.695349216461182, -53.74314498901367, -35.766883850097656, -20.570180892944336, 16.01798439025879, 48.04059982299805, 11.044171333312988, 56.299034118652344, -21.99031639099121, -0.5716305375099182, 36.650245666503906, 5.215480327606201, 28.184606552124023, 40.098304748535156, -9.279536247253418, -4.365209102630615, -18.01146697998047, -45.539207458496094, 52.7417106628418, -40.66899108886719, 22.574708938598633, -21.471181869506836, -11.107995986938477, -6.758733749389648, 16.744306564331055, -20.91006088256836, -36.48557662963867, 18.398967742919922, -19.119752883911133, 20.29947853088379, -25.268840789794922, -18.533809661865234, -40.65528106689453, -35.00834655761719, 31.444801330566406, -4.746982097625732, 13.637715339660645, 21.436031341552734, -14.251133918762207, -8.395421981811523, 12.077625274658203, 11.907197952270508, 89.57449340820312, 4.365661144256592, -7.321850776672363, -54.07953643798828, -38.53541564941406, 24.277070999145508, -9.306440353393555, -21.3287296295166, 18.58232307434082, 28.79681968688965, -36.27018737792969, -16.917192459106445, 22.123172760009766, 25.020715713500977, -7.2865142822265625, -8.402582168579102, -5.110476493835449, 25.362701416015625, -35.09321975708008, 63.37397003173828, 26.372638702392578, -50.2325553894043, -1.4529073238372803, 20.962398529052734, -10.305830001831055, 6.120352268218994, -60.61408996582031, 53.57684326171875, -25.916893005371094, -18.931581497192383, -59.643775939941406, 21.054473876953125, -16.081342697143555, -5.305782318115234, 50.254730224609375, -51.095664978027344, -1.5947600603103638, -55.91887664794922, 18.829330444335938, 7.372895240783691, 2.5608160495758057, 6.933868408203125, -19.216819763183594, 8.533458709716797, 5.090237140655518, -6.440474033355713, 25.78005027770996, 15.909093856811523, -9.438642501831055, 8.519546508789062, 33.58838653564453, 36.91836166381836, -1.0019561052322388, -3.7834408283233643, 3.820650815963745, 27.305627822875977, 13.371650695800781, -12.54321002960205, -17.811141967773438, 9.629457473754883, 23.06715202331543, 27.195026397705078, -29.38787078857422, 9.440849304199219, -16.166406631469727, 42.63250732421875, 12.358928680419922, 1.0676859617233276, -11.277215003967285, 12.928749084472656, -6.855790615081787, 27.659971237182617, -34.604949951171875, -25.701583862304688, 7.25542688369751, -3.830047845840454, -63.10469055175781, 10.043171882629395, -33.821109771728516, -6.058052062988281, -24.019563674926758, 78.6816177368164, -14.895236015319824, 28.30906105041504, -2.0286099910736084, -8.862881660461426, 27.98998260498047, -43.73286437988281, -62.3430290222168, -5.23233699798584, -29.082477569580078, -38.80967330932617, -24.405847549438477, 1.0084614753723145, -1.4408231973648071, -32.95306396484375, -24.9296932220459, 6.742460250854492, 16.633146286010742, 9.361567497253418, 27.77956771850586, -49.35943603515625, 34.76444625854492, -40.445796966552734, -42.189109802246094, 12.17872428894043, 45.16905975341797, -29.895572662353516, -11.402397155761719, -18.58587074279785, -14.659616470336914, 27.621994018554688, -30.04372787475586, -0.3446025848388672, -8.236419677734375, -31.205598831176758, 1.2745271921157837, 22.553850173950195, 29.963394165039062, 14.319131851196289, -45.03676986694336, -41.75001525878906, -26.61469078063965, -27.4714298248291, 5.224998950958252, -43.15064239501953, 31.427114486694336, 59.45599365234375, 6.995798110961914, 28.275354385375977, 13.090935707092285, -58.34174728393555, 45.40668487548828, 29.329587936401367, -49.5837287902832, 1.8136389255523682, 18.188961029052734, 9.359987258911133, 80.23664855957031, 30.40989112854004, 2.9475455284118652, -68.34576416015625, -49.761844635009766, 60.015708923339844, -41.450927734375, -38.410667419433594, 40.57326889038086, -9.03328800201416, -32.4701042175293, 13.26952838897705, 6.8371710777282715, 11.946351051330566, -23.494720458984375, 6.690418720245361, 35.6448974609375, 7.495969772338867, -20.497913360595703, 37.00007247924805, -3.7417407035827637, -41.8878288269043, 20.018722534179688, -16.701705932617188, -23.726362228393555, -1.8248374462127686, 29.707794189453125, -37.9809684753418, 47.810054779052734, 52.063568115234375, 20.55803680419922, 3.616253614425659, -40.697967529296875, 9.957870483398438, 37.886104583740234, -22.101760864257812, 6.63301944732666, 3.510556221008301, 20.4030818939209, -10.948063850402832, -10.685619354248047, 46.48884201049805, -2.4226701259613037, 21.279949188232422, -18.662796020507812, 16.464475631713867, -25.66591453552246, -10.335859298706055, -24.48051643371582, -19.168163299560547, -1.642442226409912, 7.821529865264893, -12.962994575500488, -2.9852042198181152, 31.5484676361084, -6.47996187210083, 47.19119644165039, -12.26154899597168, 35.65621566772461, -4.536849021911621, 20.46512222290039, 1.7684184312820435, 21.005760192871094, 18.973875045776367, -67.19861602783203, -26.559850692749023, -2.4533779621124268, 72.59304809570312, 3.157078504562378, 46.1279182434082, -37.9222297668457, -23.455738067626953, 42.04035949707031, 14.799138069152832, -3.212998390197754, -20.77640724182129, -11.896869659423828, -16.128602981567383, 12.32950496673584, -1.2626678943634033, -13.591874122619629, 6.088582515716553, 25.706741333007812, -50.227718353271484, 45.618247985839844, 19.276023864746094, 15.70924186706543, -30.569965362548828, 30.54375457763672, -14.013916969299316, 31.60294532775879, -23.989816665649414, -13.869152069091797, 7.881851673126221, 0.2164773792028427, 5.022930145263672, -36.17609405517578, -16.954330444335938, -30.415489196777344, -49.18210220336914, 18.72002410888672, 37.32509994506836, 12.769622802734375, -25.908924102783203, -19.789154052734375, -7.431521415710449, -12.322098731994629, 53.262264251708984, 0.6621982455253601, -27.695053100585938, 20.908864974975586, -10.982366561889648, -5.606051921844482, -7.268784523010254, 40.59904479980469, 12.3992919921875, 28.472454071044922, -13.753382682800293, 6.9084391593933105, -27.562868118286133, -28.469261169433594, -57.329322814941406, 60.8946418762207, -27.770505905151367, -24.338449478149414, -44.09374237060547, 37.633338928222656, -30.658042907714844, -16.005189895629883, -37.91205596923828, 12.54784870147705, 26.629838943481445, -26.671066284179688, 22.259721755981445, 10.797365188598633, 22.94378662109375, -18.995502471923828, -46.061676025390625, 12.830789566040039, 21.36823844909668, -61.3904914855957, 19.025354385375977, 3.317681074142456, 42.59107208251953, 25.510568618774414, -15.068306922912598, -18.344112396240234, -52.202667236328125, -27.712512969970703, -16.918947219848633, -0.22098006308078766, 42.31056213378906, 44.87499237060547, -39.072139739990234, 29.44400405883789, 19.363039016723633, -3.564493179321289, 18.781612396240234, -28.656204223632812, 12.998278617858887, 10.029133796691895, 45.13274002075195, 21.628173828125, 13.906025886535645, 22.17220687866211, 12.685720443725586, -33.44252395629883, -32.58143997192383, -15.523401260375977, -31.6118106842041, -24.9134521484375, 19.321786880493164, 65.39723205566406, 7.3212199211120605, -34.55049133300781, -18.112308502197266, -41.08775329589844, 18.509286880493164, -25.16547966003418, 36.95420455932617, -1.6537806987762451, 46.90433883666992, 51.14118576049805, -4.001689910888672, 23.014530181884766, -28.848499298095703, -15.989848136901855, -14.41744613647461, 10.442194938659668, -27.21276092529297, -86.40478515625, 1.352336049079895, 5.732865810394287, 6.475815773010254, 5.895018100738525, -16.858911514282227, 86.10845184326172, -38.33998107910156, -24.084922790527344, 55.4586296081543, 14.058394432067871, -31.198217391967773, 33.91633987426758, -22.208187103271484, -0.6943045854568481, 39.845611572265625, 29.455677032470703, 66.64820861816406, 25.487770080566406, -34.42564010620117, -0.720469057559967, -8.189587593078613, 12.278565406799316, 0.3218745291233063, -28.344419479370117, 22.852210998535156, -5.629440784454346, -0.2631854712963104, -14.431386947631836, -7.632678031921387, -67.08040618896484, -24.072845458984375, -29.287612915039062, 14.214628219604492, 16.0339412689209, 12.465777397155762, 5.368397235870361, -48.88067626953125, 43.11977767944336, -12.27151870727539, 11.52425479888916, -3.0825304985046387, 15.911971092224121, 16.528112411499023, 23.291553497314453, -11.023396492004395, -73.03636169433594, -47.075504302978516, -24.88064956665039, 15.845871925354004, 24.862834930419922, 60.97319793701172, -12.93948745727539, -39.87342834472656, -9.532642364501953, 6.203961372375488, 30.789058685302734, 18.457979202270508, -2.5406644344329834, 27.02600860595703, 21.00652503967285, 18.38914680480957, 10.339876174926758, -3.6159579753875732, -37.9989128112793, -21.03639030456543, 28.396209716796875, 10.12192440032959, 22.32265281677246, -23.269996643066406, 13.027515411376953, -2.597334861755371, 11.898923873901367, -29.27228546142578, 29.63507652282715, -6.188467979431152, -0.43966227769851685, 11.277612686157227, 10.905234336853027, -11.338848114013672, -16.279333114624023, 62.4565544128418, 15.353636741638184, -25.073638916015625, 1.973178505897522, 27.264596939086914, 63.617557525634766, -7.162914276123047, 43.405426025390625, 39.79774475097656, 32.46292495727539, -26.930261611938477, -5.376475811004639, 1.0052357912063599, 21.634441375732422, 4.352133274078369, -17.590856552124023, -7.559058666229248, 0.6102933287620544, 3.6349124908447266, 39.07243347167969, 22.31793975830078, 39.44599914550781, -28.919139862060547, -39.711429595947266, -3.6795876026153564, -48.883094787597656, -8.541258811950684, -9.111483573913574, 30.258934020996094, -27.300819396972656, -23.31490135192871, 69.88159942626953, 26.744667053222656, -18.22776222229004, -20.229223251342773, 6.766733169555664, -17.813596725463867, 15.577547073364258, -15.256057739257812, 2.0910165309906006, 38.00727081298828, -19.521684646606445, -49.84748077392578, 15.277952194213867, -28.687084197998047, 6.131997585296631, -13.072470664978027, -10.909725189208984, -40.17049026489258, -9.806086540222168, 41.99806213378906, -13.72790813446045, -24.70452880859375, 58.16226577758789, -38.14689636230469, 10.903337478637695, 12.33001708984375, 27.464706420898438, -26.42630958557129, -4.682560920715332, 6.971748352050781, 13.582284927368164, -2.8545479774475098, -26.62067985534668, -38.56306838989258, -39.05104064941406, -31.324792861938477, 21.22801399230957, 36.4875373840332, -51.50387191772461, -14.238347053527832, 21.880603790283203, 9.898383140563965, -47.27717208862305, -25.37353515625, -19.084686279296875, -38.52970886230469, -16.336650848388672, 47.21902847290039, 1.7004573345184326, 51.17513656616211, -26.342674255371094, -40.34453201293945, -47.2158203125, 5.380918979644775, 20.008522033691406, 45.787841796875, -17.806751251220703, -2.5067996978759766, -84.51464080810547, 20.064876556396484, 15.719749450683594, -14.550060272216797, -31.371591567993164, -15.45880126953125, 28.46527671813965, -38.66706085205078, -21.753616333007812, -2.341392755508423, 25.471357345581055, 46.31949234008789, 39.20070266723633, -35.36845397949219, -2.886066198348999, -15.969049453735352, 58.54292678833008, -25.96798324584961, 0.9058120846748352, -14.231821060180664, -7.997119903564453, 10.84499740600586, 5.224941253662109, -11.989948272705078, 39.64452362060547, 34.34037399291992, -10.263943672180176, -47.8438835144043, -14.162348747253418, -5.779545783996582, 9.177610397338867, -16.762962341308594, -7.56150484085083, -30.800464630126953, -4.038480758666992, -16.879682540893555, 11.836223602294922, 11.988285064697266, 64.72079467773438, 16.327512741088867, 10.057056427001953, 31.894927978515625, 28.036006927490234, -3.7126846313476562, -8.418309211730957, 47.97683334350586, -1.8928639888763428, 40.85009765625, 32.63065719604492, -18.72132682800293, -27.19055938720703, 3.4844326972961426, -47.61488723754883, -16.879131317138672, -43.66624450683594, -46.8214111328125, -15.487555503845215, 29.91428565979004, 0.5490320324897766, -53.421470642089844, 3.1659014225006104, 45.091793060302734, -26.970617294311523, 4.644369602203369, -20.422203063964844, 25.03705406188965, -12.696457862854004, -30.965734481811523, -22.57795524597168, 20.708532333374023, -12.253085136413574, 10.76832389831543, -25.908695220947266, 2.5954737663269043, 16.409481048583984, 13.044271469116211, 32.30632019042969, 7.436980724334717, -13.783899307250977, 4.108414649963379, 17.46628761291504, 1.9956603050231934, 5.7145867347717285, 9.946889877319336, 5.598090171813965, -6.20933723449707, -24.532569885253906, -3.6513302326202393, -35.7761116027832, 32.91912841796875, -5.948413372039795, -16.54656219482422, -16.806310653686523, -50.09165954589844, 12.142991065979004, 46.10947799682617, -51.38637924194336, 3.4706151485443115, -4.114537239074707, -10.63927936553955, 22.762182235717773, 5.729777812957764, 5.270705699920654, 45.930694580078125, 48.44519805908203, 8.290779113769531, -20.566478729248047, 18.562702178955078, 9.914777755737305, -29.306072235107422, -47.23365783691406, -18.30054473876953, 64.62574005126953, -30.398757934570312, 44.18492126464844, 34.06621551513672, -0.6770436763763428, -55.393333435058594, 68.08808898925781, 16.952411651611328, -5.605003833770752, 29.75539207458496, -13.39490795135498, -1.9091713428497314, -27.65066909790039, 11.616193771362305, -4.652634620666504, 40.02436828613281, 24.02883529663086, 35.3838005065918, -11.375105857849121, 2.558828830718994, 18.809537887573242, -61.40568542480469, -5.112336158752441, 16.105026245117188, 36.0166130065918, -39.4818229675293, 5.322207450866699, 40.56346130371094, 9.873170852661133, -63.703857421875, 11.956210136413574, -47.03215789794922, -15.387243270874023, -24.066444396972656, -22.16913604736328, 25.0692195892334, 41.11739730834961, -10.286346435546875, -9.525775909423828, 50.29384231567383, -26.157936096191406, 24.19267463684082, -32.250675201416016, 22.425188064575195, -17.02395248413086, -56.45401382446289, -41.33940887451172, 15.282877922058105, -44.62090301513672, 10.373357772827148, -53.65143966674805, -9.767721176147461, -55.65494155883789, -28.663665771484375, 2.6267290115356445, 37.07649612426758, 39.95514678955078, 41.17572021484375, 62.98773193359375, 22.587230682373047, 5.9165730476379395, -37.561946868896484, 15.13536548614502, -13.360806465148926, 35.675559997558594, 59.83190155029297, -8.443933486938477, -52.47642135620117, -31.245485305786133, 3.026008367538452, -0.9785833954811096, -33.0802116394043, 31.210391998291016, 3.0661473274230957, -1.6664270162582397, -5.20211124420166, -30.328105926513672, 0.4524171054363251, 59.21168899536133, 39.08426284790039, 15.937113761901855, 54.5926628112793, -1.3451027870178223, 9.591559410095215, -15.136150360107422, 13.086061477661133, 44.44489669799805, -49.642616271972656, 5.427562713623047, 21.066375732421875, -35.03213119506836, 9.0492582321167, 2.32011342048645, -30.490440368652344, -29.75455093383789, 34.64663314819336, -32.96467208862305, 6.574116230010986, 6.214293956756592, -42.88676071166992, -61.951297760009766, -25.144256591796875, 22.409564971923828, 2.863011598587036, -1.6005724668502808, 13.089152336120605, 4.6514692306518555, 7.793729782104492, -4.07265043258667, 15.658902168273926, 12.457318305969238, 21.892621994018555, 26.00664710998535, 0.07936814427375793, 46.048683166503906, 8.090293884277344, -14.408403396606445, -99.2829360961914, 66.80812072753906, -18.96714210510254, 2.0044593811035156, 13.151954650878906, 48.55744171142578, 31.628311157226562, 18.006324768066406, 34.000022888183594, 2.688326597213745, 21.68473243713379, -30.07646369934082, 4.913912296295166, -25.796451568603516, 69.95985412597656, -16.42099952697754, 40.531307220458984, -30.053071975708008, -46.9931755065918, 49.75602340698242, 9.039552688598633, -30.43962860107422, -44.33006286621094, -6.42115592956543, 2.7677319049835205, 23.391345977783203, -50.58479309082031, 16.35734748840332, -10.448387145996094, -25.83051300048828, -23.57435417175293, -0.39448606967926025, -0.8252405524253845, 2.220552921295166, 31.401168823242188, -56.49604415893555, 29.441009521484375, 17.389530181884766, -19.199939727783203, -13.38308334350586, 7.9842047691345215, 80.55891418457031, 1.5512229204177856, -64.28348541259766, 11.80191707611084, 31.29332160949707, 26.892803192138672, -40.143165588378906, 5.356441497802734, 37.93229675292969, 26.157257080078125, -1.3965867757797241, -4.23073673248291, 21.002487182617188, -7.216011047363281, -30.01283836364746, -32.00205993652344, 52.669677734375, 19.14116668701172, 30.42365264892578, -5.5803070068359375, -1.794952154159546, -48.616119384765625, 10.97819995880127, 4.898928165435791, 15.860013008117676, -13.129685401916504, -18.49339485168457, 30.00179100036621, -14.200139045715332, -10.009303092956543, -22.853912353515625, 11.402814865112305, 24.562374114990234, 7.734803199768066, 2.2555642127990723, -77.7811508178711, 24.94046401977539, -47.81692123413086, -1.2872098684310913, -36.47421646118164, 52.31498718261719, 20.435285568237305, 26.121755599975586, 11.803370475769043, 16.921680450439453, 64.7649917602539, 4.572422027587891, -65.06546020507812, -22.563827514648438, 44.46541976928711, -16.931520462036133, -19.57805824279785, 37.72849655151367, -19.66480255126953, -25.087594985961914, -6.152696132659912, -36.80025863647461, -5.719226360321045, -39.543724060058594, 18.094959259033203, 29.40354347229004, 33.767391204833984, 24.794931411743164, -13.927446365356445, -26.198972702026367, -27.865793228149414, 21.393274307250977, -3.6662917137145996, -18.699981689453125, 7.934432029724121, -22.283863067626953, 6.803108215332031, 9.204767227172852, 26.997514724731445, 31.185104370117188, -46.841461181640625, -20.690343856811523, -7.213171005249023, -44.360965728759766, 31.186302185058594, 22.538362503051758, 2.0017120838165283, 26.675018310546875, -26.48690414428711, -0.8417755961418152, 20.715953826904297, -14.974923133850098, -39.32197570800781, 11.735401153564453, -42.55535888671875, -2.0059218406677246, 14.814495086669922, 15.022891998291016, 14.927861213684082, -1.8062489032745361, 49.780704498291016, 11.825793266296387, -17.640087127685547, -12.08267593383789, 11.997953414916992, -41.02702713012695, -11.350275039672852, 28.585561752319336, 32.55599594116211, -25.629390716552734, -5.922478675842285, 29.634675979614258, 29.03984832763672, -52.23169708251953, 1.550136685371399, 8.03925895690918, -1.3935139179229736, 22.678558349609375, -15.221221923828125, -10.037139892578125, 9.358758926391602, 23.18130111694336, 14.176562309265137, -35.46809005737305, -24.059803009033203, 8.259881973266602, 8.991728782653809, -20.278425216674805, 18.702001571655273, 62.05894088745117, -22.362533569335938, -2.2412266731262207, 33.29261016845703, -15.130582809448242, -2.7829806804656982, -7.574275970458984, -6.671933650970459, -8.4490327835083, -8.86967945098877, -13.179858207702637, -14.362239837646484, -5.635989189147949, -85.04647827148438, -82.09710693359375, -11.944985389709473, -4.775583744049072, 31.843582153320312, 6.132317066192627, 13.700934410095215, -40.183528900146484, 13.564780235290527, -23.757713317871094, -55.879127502441406, 34.0350227355957, 72.21196746826172, 43.06499481201172, 10.010733604431152, -57.80275344848633, -0.29528868198394775, 23.228071212768555, -19.90188980102539, -11.518467903137207, 31.266624450683594, -14.541272163391113, -59.26957321166992, 30.96295166015625, 0.3219006061553955, -8.431885719299316, 9.018560409545898, -9.368762016296387, -7.723560333251953, 16.522598266601562, -29.257568359375, -45.823429107666016, 4.750685691833496, -29.818347930908203, -18.085350036621094, 3.6774826049804688, -25.595067977905273, 43.4307975769043, -9.263425827026367, -4.148994445800781, -40.59965515136719, -32.976966857910156, 53.77383804321289, 15.727983474731445, -21.744890213012695, -12.02149772644043, -22.802629470825195, -12.307870864868164, -34.258941650390625, -22.164247512817383, 6.913179397583008, -8.62446117401123, -26.757139205932617, 2.7290380001068115, -84.82206726074219, -13.86514949798584, 27.822843551635742, 14.674530982971191, -5.986157417297363, 0.9717986583709717, -1.515515923500061, 2.536949634552002, 12.209867477416992, -8.665403366088867, 44.136539459228516, 3.7796714305877686, -42.737823486328125, 25.64278221130371, -68.87205505371094, -24.772855758666992, 19.1074161529541, -61.09102249145508, -22.12514877319336, 21.762882232666016, 22.111936569213867, 0.23177355527877808, 14.619431495666504, 30.97953224182129, 16.511598587036133, 8.634791374206543, 19.06863784790039, -23.470359802246094, -0.08441519737243652, -15.758755683898926, -16.914878845214844, -5.650607585906982, -48.60276794433594, -22.54770851135254, 2.7476987838745117, -22.513147354125977, -27.197546005249023, 20.052207946777344, 0.7403050065040588, 21.230356216430664, 15.579475402832031, 68.84395599365234, -6.428158283233643, 6.346146106719971, -4.691728591918945, 1.8327140808105469, -25.406803131103516, 9.324723243713379, -37.32489776611328, 5.027655124664307, 8.801349639892578, 49.849525451660156, -65.90775299072266, 11.427923202514648, -53.34607696533203, -2.5082650184631348, 16.293813705444336, -8.402215957641602, -17.37625503540039, 0.021590612828731537, -27.893720626831055, 63.6479606628418, 43.22053909301758, 36.29600524902344, -10.91590404510498, 51.30131530761719, 31.5579833984375, 22.637680053710938, -25.46277618408203, 33.940147399902344, -26.393545150756836, 61.03561782836914, -19.38003921508789, -26.687244415283203, 39.36152648925781, 14.08834457397461, -19.12030029296875, -0.6854425668716431, -25.608041763305664, 7.496710777282715, 16.31295394897461, 29.697811126708984, -14.384000778198242, -11.788283348083496, 25.012723922729492, -24.81940269470215, 21.12213897705078, 7.168704032897949, 1.3682483434677124, 43.693572998046875, -31.50284194946289, -2.3097896575927734, 50.2062873840332, -14.064253807067871, 14.354711532592773, -18.117136001586914, 29.982744216918945, 6.411810398101807, -3.0785579681396484, 6.541764259338379, -12.466829299926758, -0.5871163010597229, 5.809016227722168, 37.717681884765625, -58.500343322753906, 48.248085021972656, -27.116226196289062, -31.734745025634766, 31.74421501159668, 46.91004180908203, 3.5528838634490967, -13.716301918029785, 20.852622985839844, 14.297835350036621, 11.08945369720459, 11.659002304077148, -20.572317123413086, -27.094697952270508, 4.071380138397217, -22.542457580566406, -33.593650817871094, -29.630107879638672, 17.209938049316406, -55.20479202270508, 12.827685356140137, -75.27332305908203, -54.898780822753906, -1.2501780986785889, -11.300119400024414, -5.672077178955078, 33.326499938964844, 28.06376838684082, -1.2951189279556274, 14.89344310760498, 12.762382507324219, -33.95556640625, 21.03666114807129, -39.81394577026367, 31.59453010559082, -11.267596244812012, -33.63967514038086, 48.11900329589844, 8.110898971557617, 4.451355457305908, 8.707611083984375, 22.12871742248535, 46.560977935791016, -23.299238204956055, -18.560815811157227, -59.48630142211914, -18.572677612304688, -32.21526336669922, -36.85565948486328, -10.879281044006348, 50.72782516479492, 3.7327868938446045, -4.041153430938721, 21.290294647216797, 3.680777072906494, -17.971364974975586, -7.559828281402588, 18.07773208618164, 26.201889038085938, -50.59931564331055, -7.0864715576171875, 50.89266586303711, -21.42203712463379, 22.89992904663086, -50.60081100463867, 14.176046371459961, -12.973388671875, -23.14779281616211, 4.119022369384766, -17.135913848876953, -0.4632924497127533, -66.12947845458984, 26.363601684570312, 2.9366486072540283, 37.59731674194336, -31.251365661621094, 1.5180991888046265, -35.01348876953125, -12.509156227111816, 39.36301040649414, 16.87620735168457, 26.436269760131836, -3.7519893646240234, -52.59307098388672, 3.623326301574707, 73.85713958740234, -16.153928756713867, -5.171831130981445, 8.957283020019531, 7.706270694732666, 56.498809814453125, -21.54414939880371, -44.10277557373047, 13.88868522644043, 5.1316657066345215, -1.5213477611541748, 16.631086349487305, -22.82793617248535, 7.9842424392700195, 56.85560989379883, -18.235477447509766, -21.042253494262695, -24.128284454345703, 44.22496795654297, -7.981122970581055, -34.817283630371094, 18.196123123168945, -60.49924850463867, 24.06999397277832, -21.600994110107422, -17.354846954345703, -8.699317932128906, 64.90493774414062, -0.022180715575814247, 0.5490257740020752, -2.561096668243408, -44.89933395385742, 11.239867210388184, -31.132661819458008, -15.337239265441895, -12.354068756103516, -36.928016662597656, -11.399561882019043, -34.6484260559082, 11.545990943908691, 17.771240234375, 33.28077697753906, -66.64083099365234, -5.777468681335449, -18.704010009765625, 11.739287376403809, 9.381343841552734, 49.16737747192383, 22.03102684020996, -5.182127952575684, 28.827239990234375, -28.721847534179688, 17.809226989746094, -7.670562744140625, 14.514565467834473, 48.95299530029297, -38.15190887451172, 47.84473419189453, 38.65250778198242, 16.50535774230957, -24.273117065429688, -30.689098358154297, 26.62523651123047, -13.564046859741211, 28.05145835876465, -32.95295715332031, 9.872547149658203, 26.503849029541016, -68.20925903320312, 35.54743957519531, -16.53903579711914, -47.49602508544922, 46.4153938293457, -37.98883056640625, -15.338996887207031, 16.560972213745117, -18.41646385192871, -10.324188232421875, -22.2569637298584, 10.259181022644043, 39.454105377197266, -5.552402019500732, 0.405508428812027, -16.531715393066406, 10.203544616699219, -98.21234893798828, -9.671249389648438, 4.8212890625, -20.724349975585938, 30.452016830444336, -11.212715148925781, -23.905879974365234, 10.468891143798828, -12.480731964111328, 7.561678409576416, -29.330434799194336, -0.39509278535842896, -13.346681594848633, 3.801886796951294, 45.76380157470703, -23.143346786499023, 33.15404510498047, 22.9373779296875, 29.412384033203125, -27.79079246520996, -49.65446853637695, -30.36686134338379, -9.953827857971191, -34.918235778808594, 33.39751434326172, -8.27676773071289, 48.59857940673828, 48.971839904785156, 16.406957626342773, -34.81123733520508, 26.909055709838867, -25.164493560791016, 31.682323455810547, -33.2639045715332, 7.470211029052734, 11.877729415893555, 46.23322296142578, -62.143333435058594, 36.62108612060547, -3.761063575744629, -7.789010524749756, -46.02831268310547, -29.716964721679688, -26.748666763305664, 35.97722244262695, 14.057047843933105, -38.427520751953125, 52.70203399658203, 1.9455622434616089, -33.792503356933594, -3.3931286334991455, -9.587043762207031, 26.394119262695312, -21.105119705200195, 51.476444244384766, -36.86522674560547, 55.9248161315918, -15.282976150512695, -22.3798828125, -13.125751495361328, 36.52389144897461, 39.65428161621094, -16.197296142578125, 37.18354034423828, -45.12692642211914, -17.01604652404785, 17.505945205688477, 7.8618693351745605, -7.689664363861084, -29.207021713256836, -33.66276550292969, 8.74515151977539, -10.767509460449219, -2.1947031021118164, 4.367219924926758, -55.57899856567383, 21.671733856201172, 11.151463508605957, -52.00221633911133, 27.52088165283203, 25.94356346130371, -4.777883529663086, 9.17253589630127, 5.891417503356934, -15.88414192199707, 16.665245056152344, 28.68412971496582, 16.919172286987305, -5.407928943634033, 20.21709632873535, 21.268753051757812, -20.9803409576416, -17.610424041748047, 4.7177557945251465, -14.970743179321289, 5.730168342590332, -15.635841369628906, -20.600126266479492, -1.478588342666626, -2.867215633392334, 4.142160892486572, -41.89020919799805, -4.334216594696045, 5.9942216873168945, 14.694684028625488, -2.7419707775115967, -17.865739822387695, 10.440093994140625, -14.181806564331055, 61.42613220214844, 85.15120697021484, -42.314754486083984, -35.81769943237305, 18.662202835083008, -28.033124923706055, 58.43016815185547, 9.403615951538086, 3.27047061920166, -57.146183013916016, 27.157346725463867, 0.15613414347171783, 34.0510368347168, 14.841557502746582, 1.9800529479980469, -49.933837890625, -10.37969970703125, 15.18635082244873, 33.683841705322266, -17.157594680786133, -3.1417644023895264, -10.378279685974121, -24.97450828552246, 14.312159538269043, -48.63117980957031, 20.139904022216797, -65.60060119628906, -7.227722644805908, -35.184410095214844, -5.521563529968262, -5.447704792022705, 28.680679321289062, 7.341372966766357, 47.38164138793945, -12.984261512756348, -7.9748663902282715, 11.184289932250977, 10.547285079956055, -42.68836212158203, 51.487308502197266, 20.319345474243164, 6.2786407470703125, -2.96598482131958, -0.47898581624031067, 1.9941298961639404, 32.83533477783203, -14.760058403015137, -4.686861515045166, 58.665645599365234, 3.1610360145568848, -15.915637969970703, -46.410858154296875, 20.21226692199707, -32.657745361328125, 16.95116424560547, 35.478214263916016, -6.525068759918213, -23.01401138305664, 31.40977668762207, 0.03118002973496914, -27.54640769958496, -34.36142349243164, 12.798770904541016, -34.71584701538086, 4.2363786697387695, -14.290616989135742, -12.721504211425781, -11.464659690856934, 15.861235618591309, -13.012264251708984, 20.522708892822266, 47.89448928833008, 5.810057640075684, 61.75190353393555, -19.33360481262207, -29.792503356933594, 27.12859535217285, 27.696495056152344, -18.13718605041504, -14.287741661071777, 2.116537094116211, -5.677514553070068, 51.85783767700195, -39.32803726196289, -2.295579671859741, -38.241241455078125, -11.47472858428955, 33.67067337036133, -12.988344192504883, 34.99531555175781, 33.66548156738281, 18.12934684753418, 7.08638858795166, -47.396690368652344, 24.920024871826172, 39.19316482543945, -43.20736312866211, 41.29541015625, 21.91130256652832, 32.5562744140625, 18.857454299926758, -21.04930305480957, -33.53997802734375, -10.766508102416992, 27.14703369140625, 4.785923480987549, -58.42585372924805, 11.127606391906738, -32.182437896728516, 23.638160705566406, 30.575037002563477, -3.2745296955108643, 28.682254791259766, -28.453956604003906, -21.99869728088379, -4.299255847930908, 84.80050659179688, 45.375404357910156, -33.7533073425293, 14.216087341308594, 46.480777740478516, 16.29566192626953, -2.0287721157073975, -17.681060791015625, -22.755847930908203, -18.861921310424805, 19.02202796936035, -53.17490005493164, 34.820865631103516, 23.15277671813965, -56.26657485961914, 16.975133895874023, -13.410848617553711, 26.07961082458496, 20.726749420166016, -65.64458465576172, 6.776335716247559, 10.689138412475586, -46.4586067199707, -10.211480140686035, -24.911413192749023, 53.5101432800293, -18.06409454345703, -9.627742767333984, -15.021275520324707, 6.22414493560791, -6.136125564575195, -36.71110916137695, -30.93874168395996, -14.495378494262695, -50.32099151611328, -11.413433074951172, -29.855188369750977, 31.015565872192383, 22.204771041870117, -100.75069427490234, -4.369398593902588, -24.21104621887207, -27.496049880981445, -51.25673294067383, 13.44448184967041, 0.7640753984451294, -61.93374252319336, 5.420820713043213, -22.768648147583008, 21.53582191467285, -8.83675765991211, 24.250507354736328, 18.290555953979492, -47.375938415527344, 27.084115982055664, 16.669021606445312, 40.53660202026367, -15.705395698547363, 44.01972961425781, -12.019502639770508, -13.051481246948242, 21.072635650634766, 44.591060638427734, -19.204509735107422, -27.624900817871094, 64.1124496459961, 51.51761245727539, 1.2338528633117676, -60.84463882446289, 0.23899251222610474, -9.492880821228027, -13.526955604553223, -21.293519973754883, -41.138057708740234, 27.11396598815918, -20.052827835083008, 46.91495132446289, 13.81308364868164, 27.52994155883789, -4.024877548217773, -38.96240997314453, 4.813210487365723, 41.10434341430664, 27.679597854614258, 18.87618637084961, -5.222949981689453, 48.77392578125, -41.87504577636719, -16.267566680908203, -4.233573913574219, 8.5324068069458, 2.5076513290405273, -26.157804489135742, -52.342403411865234, 9.242631912231445, 32.31401824951172, -0.16042280197143555, -7.978158473968506, -20.727190017700195, -3.60111141204834, 22.6599178314209, 24.474964141845703, 74.59098052978516, -36.17142105102539, 8.819930076599121, -41.244110107421875, 57.15131759643555, -13.358280181884766, -23.198884963989258, 22.225940704345703, -9.815340995788574, -10.675411224365234, -16.853534698486328, 34.22374725341797, -38.77749252319336, -3.318598747253418, -8.922106742858887, -10.840778350830078, 37.336151123046875, 21.94805908203125, 5.718559265136719, 28.756746292114258, -3.736886739730835, -6.787206172943115, 30.635866165161133, -9.31647777557373, 21.316059112548828, 13.27509880065918, 3.649010181427002, -45.10991668701172, 14.557108879089355, -1.170453429222107, -14.145397186279297, -23.231597900390625, -0.4120367169380188, 6.246222496032715, 10.63294506072998, 43.91204071044922, 41.41584777832031, 12.981276512145996, -22.648929595947266, 1.4066424369812012, 11.431567192077637, -28.440595626831055, -57.907928466796875, -18.112220764160156, -12.526193618774414, -42.90633010864258, -72.75882720947266, 39.79106903076172, 7.181480884552002, -45.37740707397461, -0.05159895122051239, 46.22987365722656, 16.15773582458496, 31.119543075561523, 42.8033561706543, 28.253767013549805, -41.27919006347656, 23.56041145324707, -17.795129776000977, 10.726052284240723, -6.073078632354736, 20.55609703063965, -9.247045516967773, 18.592615127563477, 18.585521697998047, 1.6894969940185547, -8.774310111999512, 8.934215545654297, -11.604816436767578, -9.767046928405762, 15.576009750366211, 0.46818581223487854, -20.924278259277344, -6.641046047210693, 20.091041564941406, 40.8843994140625, -19.539888381958008, 33.11470031738281, -21.689611434936523, -25.237110137939453, 4.79942512512207, -3.972771644592285, -8.078960418701172, -14.570218086242676, -1.7298684120178223, -11.621472358703613, 9.135529518127441, 17.882532119750977, 14.198786735534668, -27.86749267578125, -12.025506019592285, 0.8124508261680603, -1.2371947765350342, -16.282695770263672, 13.531947135925293, 66.94258880615234, -15.30897045135498, -1.8510301113128662, -72.19329071044922, -22.48778533935547, -4.104432582855225, -7.330456256866455, -37.84373092651367, -34.00606918334961, 53.1898078918457, -30.083280563354492, -51.3876838684082, -3.294234037399292, -0.26024317741394043, -25.83393096923828, 62.44538879394531, 11.259093284606934, 20.552631378173828, 32.614139556884766, -33.12191390991211, -6.335976600646973, 2.508239507675171, 63.67558288574219, -15.26441764831543, -2.657545804977417, -17.8341007232666, 28.276765823364258, -4.344721794128418, 41.72891616821289, 30.98431396484375, 31.460817337036133, -15.389192581176758, -4.317382335662842, 13.110509872436523, -30.125391006469727, -37.711036682128906, 6.230209827423096, -5.554585933685303, 65.16898345947266, 2.0856690406799316, -6.061829566955566, -13.681676864624023, 31.059791564941406, -9.820576667785645, 8.717995643615723, -11.813558578491211, -45.87029266357422, -0.6972600221633911, 28.701797485351562, 4.431502819061279, -7.1895365715026855, -49.7357063293457, 20.58786392211914, -23.25236701965332, -27.41448211669922, 54.60757827758789, 12.979583740234375, -12.744975090026855, -8.490915298461914, -12.608878135681152, 45.204383850097656, -38.112586975097656, -2.629558801651001, 9.996270179748535, -1.2189176082611084, -9.63164234161377, -4.345668792724609, 4.095809459686279, 2.488102674484253, 19.384445190429688, -8.123394012451172, 58.57231521606445, 23.936153411865234, -32.29619598388672, -34.853858947753906, 13.394168853759766, -41.398170471191406, 29.920705795288086, 27.41312026977539, -0.6967908143997192, -63.72749710083008, 4.514767169952393, 74.05099487304688, -18.00602149963379, 24.25748062133789, 12.068859100341797, 4.70771598815918, 14.655903816223145, 31.054994583129883, 3.358550548553467, 19.615406036376953, -0.2863142192363739, -46.94312286376953, 11.1619873046875, -33.85763168334961, -21.80849266052246, -18.746307373046875, -19.04084587097168, -67.19702911376953, 16.137521743774414, 58.34772491455078, -56.441368103027344, 1.3816988468170166, -65.73702239990234, -55.453125, -31.776878356933594, -34.21521759033203, -31.41731071472168, -37.87596893310547, 40.00358581542969, 36.80268478393555, -15.669113159179688, -31.107791900634766, -8.858860969543457, 17.588808059692383, 26.153295516967773, 4.812095642089844, 21.097061157226562, 23.12004852294922, -22.350786209106445, 47.11924743652344, 12.197111129760742, 12.927388191223145, 3.6505861282348633, 27.23261833190918, 34.97785949707031, 2.8368754386901855, -52.33688735961914, 2.3667213916778564, -16.40390968322754, -56.39433288574219, -48.96962356567383, 54.0528678894043, -5.772976398468018, 2.748464345932007, -21.066545486450195, 27.362079620361328, 125.85037994384766, -30.37833023071289, 9.21790885925293, 9.553850173950195, -30.592002868652344, -26.235639572143555, -8.275300025939941, 19.1763916015625, -29.338180541992188, 9.25432014465332, -7.072904109954834, 59.64772033691406, -15.290687561035156, 57.095664978027344, 39.37287902832031, -27.396984100341797, -49.062278747558594, 39.93897247314453, 21.221914291381836, -32.96448516845703, 76.01062774658203, -16.999231338500977, 20.386438369750977, -33.62050247192383, -38.5119743347168, 27.412267684936523, 64.69420623779297, 34.04590606689453, 20.91551971435547, -3.0533390045166016, -58.98575210571289, -13.557615280151367, 32.74605178833008, -1.428367257118225, 6.910572528839111, -48.175994873046875, -2.4109690189361572, -14.0121431350708, 9.029695510864258, 2.593346118927002, 37.57065963745117, -28.21824836730957, 57.14838790893555, -34.05735778808594, 43.069007873535156, -19.990230560302734, -34.98964309692383, 11.674469947814941, -50.90691375732422, 12.508902549743652, 33.88544845581055, 28.528934478759766, -34.8095703125, 10.797407150268555, 11.723739624023438, -18.62856101989746, 27.425405502319336, 14.547746658325195, 66.7497329711914, -25.056625366210938, 9.083308219909668, 24.71595001220703, -19.211286544799805, 19.275840759277344, 6.363434314727783, 50.631614685058594, -7.001490592956543, 13.223133087158203, 32.165977478027344, -26.136272430419922, -11.443196296691895, -14.498101234436035, 38.98222351074219, 3.840820074081421, -6.814765453338623, -7.130914211273193, -1.8437907695770264, -0.4420483708381653, 1.838847279548645, 51.1719970703125, -10.543163299560547, 10.51235294342041, 10.706990242004395, 4.922120571136475, 0.08899770677089691, 5.655993461608887, 37.689517974853516, -11.049517631530762, -6.093740463256836, 1.230678915977478, -38.53882598876953, -18.6369686126709, 44.306549072265625, -34.74051284790039, -19.432910919189453, 17.937997817993164, 35.939109802246094, 7.904900074005127, -22.414127349853516, -22.754039764404297, 10.015337944030762, -32.66999816894531, -37.889305114746094, 14.646749496459961, -13.000625610351562, -58.70375442504883, 15.680805206298828, 29.151342391967773, 25.434612274169922, 61.68314743041992, 30.243602752685547, -57.20802688598633, 43.30830383300781, 5.961306571960449, 47.112911224365234, -72.0169448852539, -72.01976776123047, 18.70329475402832, -34.65040969848633, -50.6365966796875, -48.61156463623047, 8.879268646240234, 79.63121032714844, -8.262340545654297, -35.660640716552734, 11.650895118713379, 27.314851760864258, 24.750347137451172, 28.866249084472656, -27.42377281188965, 31.567157745361328, -3.9195146560668945, -39.57008361816406, 15.349382400512695, -1.2190195322036743, 16.766620635986328, -38.92997360229492, 1.3938201665878296, 11.581655502319336, 45.57136154174805, 1.8812932968139648, -24.5102481842041, 13.937283515930176, 9.016829490661621, -6.047969341278076, 13.14380168914795, 4.369574546813965, -42.38379669189453, 39.12150192260742, -18.68679428100586, -2.5459635257720947, 74.20958709716797, -24.566984176635742, 17.393823623657227, 5.3272552490234375, -30.981658935546875, 9.768305778503418, -74.96984100341797, 5.927220821380615, -23.362438201904297, 24.76335906982422, 61.0103759765625, 63.450172424316406, 47.99028778076172, -5.589761734008789, -29.498188018798828, -18.72574234008789, -5.669027805328369, 4.749952793121338, 0.8815099000930786, -3.478484630584717, -1.5512274503707886, 1.5783376693725586, -58.04825973510742, -9.283493041992188, 23.994327545166016, -27.289777755737305, 5.769105911254883, -31.361875534057617, 11.81053352355957, 20.042465209960938, 25.36304473876953, 45.70368957519531, -21.81964111328125, -7.4806952476501465, 1.3531702756881714, -5.141263961791992, 3.2501678466796875, 2.5887768268585205, 0.3111215829849243, -8.563619613647461, 2.376401662826538, -25.72901725769043, -17.946435928344727, -47.57571029663086, 39.66812515258789, 48.77872085571289, -34.34865188598633, 35.885398864746094, -53.73591995239258, 32.33224868774414, -9.809918403625488, -81.10035705566406, -11.764235496520996, 20.47242546081543, 49.24794387817383, 10.052492141723633, -25.485027313232422, 55.63871383666992, 2.9358582496643066, 20.70865249633789, 17.368127822875977, -22.400531768798828, 29.023380279541016, 18.962636947631836, -7.460946083068848, -13.473014831542969, 18.375499725341797, -27.1136417388916, 2.7325682640075684, 23.548351287841797, -0.004388274624943733, -13.31276798248291, -7.847276210784912, -40.52217483520508, -27.446876525878906, -43.9338493347168, -46.39276123046875, -41.77475357055664, -35.329925537109375, -35.82931137084961, 7.696600437164307, -11.413686752319336, -21.102813720703125, 12.339937210083008, 4.539135932922363, -7.388996124267578, 6.330214023590088, -10.396062850952148, 45.18444061279297, 26.340028762817383, -30.07887077331543, -11.664959907531738, 4.163657188415527, 5.56318998336792, 34.43400192260742, -5.393091201782227, 5.410050868988037, 28.324419021606445, -63.76610565185547, -31.419099807739258, 31.84474754333496, 18.411062240600586, -41.3430061340332, -22.356746673583984, -26.470096588134766, -23.360774993896484, 24.166915893554688, 59.360565185546875, -31.345951080322266, -41.63809585571289, 44.24103927612305, 38.82635498046875, -38.390445709228516, 62.413822174072266, 12.016990661621094, 16.916624069213867, 66.70665740966797, -31.38782501220703, -25.69806480407715, 27.73312759399414, 25.96342658996582, 15.82529067993164, 108.73497009277344, 7.644742012023926, 48.96147155761719, 18.572349548339844, -5.990061283111572, 2.2917041778564453, 66.85933685302734, -3.661233425140381, -3.2967681884765625, -38.83402633666992, -19.6234130859375, -21.66045379638672, 16.475122451782227, -23.531509399414062, -16.8758544921875, -15.224533081054688, -50.30819320678711, -18.67667007446289, 18.510948181152344, 29.54277992248535, 12.050106048583984, 16.11058235168457, -37.618831634521484, -12.378129005432129, -18.60325813293457, 5.235685348510742, 25.518836975097656, -45.92837905883789, 3.3743839263916016, -56.31863021850586, 55.14142608642578, 12.124842643737793, -18.681636810302734, 21.14853286743164, -9.650838851928711, 9.23441219329834, -1.9712779521942139, -40.44687271118164, -9.199151039123535, 17.92506980895996, 21.058103561401367, 57.36236572265625, 16.998138427734375, -53.31754684448242, 15.772723197937012, -10.231485366821289, 20.475688934326172, 18.624530792236328, -18.816078186035156, -55.936553955078125, -37.94752883911133, 45.99264907836914, 1.2447155714035034, -27.742902755737305, -4.191464424133301, -6.539361476898193, 59.987586975097656, 10.49724006652832, -19.655309677124023, -20.71575927734375, -31.983840942382812, 8.342724800109863, 42.48823928833008, 9.65983772277832, -9.084410667419434, -16.209836959838867, -5.292888641357422, -10.088417053222656, -35.61012268066406, -39.510101318359375, 37.477760314941406, -18.578798294067383, 14.660796165466309, 61.69044876098633, -21.954679489135742, -15.714859008789062, -18.523183822631836, -52.779911041259766, 80.54525756835938, 29.087289810180664, 6.10960054397583, 23.853845596313477, 37.984371185302734, -39.73931884765625, -29.353322982788086, 11.225761413574219, -2.2944233417510986, -27.33129119873047, -6.782452583312988, -62.561180114746094, -15.662140846252441, 38.62337112426758, 34.31196594238281, 27.706161499023438, 6.851984977722168, -45.04805374145508, -55.204586029052734, 9.881369590759277, 30.249588012695312, 31.488876342773438, 35.780250549316406, -18.965665817260742, -20.002960205078125, 1.0830010175704956, -43.013614654541016, -39.37068557739258, 12.909920692443848, -21.569461822509766, 73.4354248046875, -34.21733093261719, -38.01982879638672, 6.667646884918213, -35.69393539428711, 32.26856231689453, -2.2005841732025146, -28.432329177856445, 0.622550904750824, -82.02481079101562, -20.472976684570312, -21.264366149902344, 34.10795211791992, 6.684988021850586, 37.907249450683594, -46.926246643066406, 42.278228759765625, -7.4387617111206055, -6.619708061218262, 40.13375473022461, 38.69403076171875, 39.14654541015625, -80.73873901367188, -6.974498748779297, 56.972320556640625, -2.315699338912964, -39.746910095214844, -17.71051597595215, -30.049163818359375, -23.007617950439453, 22.01499366760254, 13.091029167175293, 42.205406188964844, 54.868896484375, -37.764976501464844, 17.969615936279297, 36.657474517822266, -89.22747802734375, 3.2953925132751465, 17.817644119262695, 26.741199493408203, 57.84122085571289, -4.573183059692383, 0.8678749203681946, -1.8704888820648193, 28.08582305908203, 22.3132381439209, -40.547828674316406, -13.8608980178833, 33.782161712646484, -42.51020431518555, 67.73978424072266, 16.386749267578125, 25.351999282836914, -20.12400245666504, -6.221805572509766, 13.81847095489502, -0.8846405744552612, 2.4529268741607666, -9.157135963439941, 0.19535031914710999, -36.067256927490234, 22.107894897460938, 4.352964878082275, -11.440747261047363, 0.5764104127883911, 1.5383639335632324, -32.356929779052734, -0.9420688152313232, 22.267330169677734, -11.067331314086914, 8.078278541564941, 13.557136535644531, -23.269859313964844, 19.384891510009766, 22.96961212158203, 39.70347595214844, -8.527044296264648, -7.754261016845703, -27.733428955078125, 27.938819885253906, 14.703176498413086, 9.06709098815918, 18.371475219726562, 27.62133026123047, 40.57474136352539, -2.0685839653015137, 47.19851303100586, -4.87237024307251, 63.445980072021484, 2.5379230976104736, 15.111197471618652, 21.239233016967773, -17.397174835205078, -50.20989227294922, 45.34064483642578, 67.33618927001953, 26.788820266723633, 44.59027862548828, 15.466107368469238, -19.00423240661621, -18.437334060668945, 23.01116371154785, 36.93873977661133, -7.401764869689941, -21.769489288330078, -9.522756576538086, 1.3310853242874146, -65.06449127197266, 36.14606475830078, -44.5506477355957, -29.16205406188965, 8.894684791564941, -8.362676620483398, -46.06157684326172, -41.07901382446289, -37.013553619384766, 60.777259826660156, -39.28129959106445, -36.22343063354492, 38.91858673095703, 16.84052085876465, 10.400733947753906, -49.33021926879883, -55.479888916015625, -14.526637077331543, -9.277826309204102, 30.95527458190918, 40.58832931518555, 6.984877109527588, 6.148345470428467, 59.49516296386719, -17.283193588256836, -41.84478759765625, -37.235477447509766, 24.282270431518555, -16.768314361572266, -13.55065631866455, -42.994422912597656, 41.285491943359375, -42.556888580322266, -1.4046825170516968, 8.576412200927734, 18.6640567779541, -21.74079132080078, -11.057493209838867, -9.104243278503418, 2.3292734622955322, -16.38630485534668, 37.88745880126953, 70.3768310546875, 2.368457555770874, 10.661880493164062, -2.281646728515625, 52.6797981262207, -28.0892276763916, 26.989059448242188, 6.824844837188721, 22.219135284423828, -20.942235946655273, -15.19955825805664, -19.21101951599121, -14.304264068603516, -3.0453805923461914, -63.86630630493164, 8.252598762512207, 14.466785430908203, 20.98099136352539, -24.119577407836914, 15.664175987243652, -16.061227798461914, 6.997818470001221, 17.76751136779785, 26.59819221496582, 58.31024932861328, -17.30550193786621, -23.140140533447266, -50.34080505371094, 0.6401147246360779, 136.46075439453125, -43.4940185546875, -4.418041229248047, 17.20016098022461, -15.66976547241211, 47.67885208129883, -18.893037796020508, 4.4742021560668945, -32.432579040527344, -36.86460494995117, 20.873624801635742, 22.41307258605957, -21.44222068786621, 15.555365562438965, 4.941004753112793, 22.122514724731445, 16.068620681762695, -6.077710151672363, 29.72836685180664, -57.832733154296875, -51.88436508178711, -11.269979476928711, 39.816925048828125, 43.32429885864258, -21.859779357910156, -3.566488742828369, -35.81185531616211, -2.4577648639678955, -51.38531494140625, -4.489799499511719, 8.473381042480469, -54.7859992980957, -31.0002498626709, 25.940961837768555, 8.506564140319824, -45.16327667236328, -5.706521511077881, 22.093515396118164, -41.68013381958008, 15.119372367858887, 3.3127517700195312, 1.372591257095337, 34.29631423950195, -25.87773323059082, 11.764572143554688, 41.29026412963867, -3.834414482116699, -14.368119239807129, -14.541353225708008, 8.517693519592285, -4.655031204223633, -23.22420310974121, -20.641611099243164, 30.008203506469727, 8.498985290527344, 47.579750061035156, 48.129451751708984, 41.70539093017578, -12.59946346282959, 14.468289375305176, 31.975608825683594, -3.8349504470825195, -6.208248138427734, -17.922475814819336, 40.35624694824219, -2.555572271347046, 33.03342819213867, -16.826412200927734, -44.77601623535156, 10.849813461303711, -41.61590576171875, -30.53058433532715, -65.40929412841797, -49.58173751831055, -6.221565246582031, -15.059221267700195, -18.64699935913086, 12.043228149414062, 0.4495966136455536, -7.643623352050781, -11.635126113891602, 51.06940460205078, 26.1889591217041, -30.147232055664062, -19.352222442626953, -41.50473403930664, -8.01681137084961, -15.663534164428711, -3.281214475631714, 31.92624855041504, -1.333715558052063, 19.212499618530273, -16.444717407226562, -6.536764144897461, -11.867700576782227, 38.50578689575195, -27.287151336669922, 60.26713943481445, 10.165103912353516, -28.225263595581055, -25.897571563720703, -7.5334577560424805, -12.140888214111328, -1.2286370992660522, -7.1549787521362305, 21.133283615112305, -57.008392333984375, -16.926931381225586, 14.592138290405273, 0.9085778594017029, -37.32334899902344, -3.1704323291778564, -39.5941162109375, -32.40580368041992, 15.62661075592041, -29.65582275390625, 13.684765815734863, 9.021039962768555, 21.135515213012695, -14.335548400878906, -26.46466064453125, 0.5311669111251831, -17.48921775817871, -32.0410270690918, -52.308414459228516, -43.022804260253906, 31.226696014404297, 37.10658264160156, -19.531572341918945, 34.3282470703125, 4.367664813995361, -6.377209186553955, -27.44744873046875, 5.476504325866699, -0.8034404516220093, 10.85807991027832], [-3.762625217437744, -22.707277297973633, -16.7199764251709, -31.19468116760254, 17.30047607421875, 57.4132080078125, -30.812599182128906, 33.38393783569336, -12.014058113098145, -11.715168952941895, -28.98236846923828, 9.357666015625, 24.12312889099121, 15.34768009185791, -15.35730266571045, 19.687772750854492, -37.907508850097656, -38.57310485839844, -12.987078666687012, -1.9264999628067017, -24.544631958007812, 18.111358642578125, -6.652375221252441, -11.574504852294922, -5.267965316772461, -3.1946773529052734, -44.22207260131836, 7.495222091674805, 72.45797729492188, 13.852972984313965, 4.711499214172363, 13.036664009094238, -3.634650707244873, 0.8444465398788452, 27.960678100585938, -14.89693832397461, 76.13121795654297, 87.45248413085938, 4.764915466308594, 23.507902145385742, -45.35170364379883, 20.733375549316406, -57.173404693603516, -40.03207778930664, 3.2595131397247314, -11.737929344177246, -10.378292083740234, 4.622900009155273, 8.723670959472656, 34.92103958129883, 99.96636962890625, -11.66908073425293, -6.949881076812744, 4.080794811248779, 16.721261978149414, 31.71942138671875, 45.095726013183594, 14.825522422790527, 49.38459396362305, 20.2475643157959, -12.401902198791504, -25.260665893554688, -54.660343170166016, -34.06196212768555, -70.52651977539062, 43.51222229003906, 3.66414475440979, 0.48888009786605835, -6.96994161605835, -11.985706329345703, -78.55807495117188, 61.8828125, 68.55792999267578, -48.7296142578125, -29.65406036376953, -14.860647201538086, 31.630496978759766, 33.201107025146484, 17.412879943847656, 18.10948944091797, 19.12458610534668, -29.44141387939453, 23.642017364501953, 16.66678810119629, 29.407583236694336, 1.1595048904418945, -18.47883415222168, 33.81447982788086, 48.26993942260742, 22.713031768798828, 44.16678237915039, -9.689508438110352, -21.912872314453125, -36.74830627441406, 8.705565452575684, -32.98123550415039, 17.04342269897461, 27.659648895263672, -20.584640502929688, -18.72703742980957, 18.764066696166992, 47.42780303955078, 28.637508392333984, -10.291184425354004, 0.7243552207946777, 68.8304672241211, -62.48091506958008, 55.75904083251953, -4.528194427490234, -17.866806030273438, -38.68888854980469, 25.1503963470459, 38.704647064208984, -18.72473907470703, 4.223628044128418, -10.61557388305664, 49.2559928894043, 48.57299041748047, -41.02918243408203, -14.947613716125488, -4.010809898376465, -7.567227840423584, 32.38936233520508, 8.220949172973633, -32.70102310180664, 5.525010108947754, 3.723064661026001, -20.51312255859375, 27.49625015258789, -12.09972095489502, -12.067771911621094, 55.62803649902344, -5.072844505310059, -52.445838928222656, 2.712207794189453, 31.853788375854492, -0.4616902768611908, -5.879539489746094, -38.23153305053711, 2.4901175498962402, -38.870452880859375, -28.628000259399414, 8.206867218017578, 11.738395690917969, 6.074467658996582, -25.906877517700195, 23.181438446044922, 34.57762908935547, -5.833247661590576, -51.1616096496582, -6.868892669677734, -41.42426300048828, 11.944626808166504, 59.133941650390625, -4.798709869384766, -64.0410385131836, 23.278392791748047, -3.8160057067871094, 32.32378387451172, -36.844932556152344, 33.784759521484375, -27.382217407226562, 0.8508309721946716, -32.18913650512695, 10.806437492370605, 28.002574920654297, 2.1997132301330566, 58.764530181884766, 25.255287170410156, -3.559567928314209, -27.31229591369629, -21.7838077545166, -10.383099555969238, 49.47855758666992, 31.264698028564453, -30.35240936279297, 21.615297317504883, 10.48206615447998, -50.425445556640625, -20.35485076904297, 49.12261199951172, 13.234719276428223, 0.6021859645843506, -14.057790756225586, -12.601997375488281, -17.21232032775879, 35.76667785644531, 33.31688690185547, -13.592812538146973, -42.305912017822266, -3.881969928741455, -17.260295867919922, 18.4899959564209, -1.9802979230880737, 1.5099416971206665, -2.642957925796509, -32.78480911254883, 27.94573211669922, -43.981605529785156, -20.574256896972656, -16.900524139404297, -41.30443572998047, -48.93809509277344, 20.892139434814453, 83.54985809326172, 29.171411514282227, -15.296725273132324, -30.195449829101562, -5.691821575164795, 43.48328399658203, -9.967999458312988, 38.566184997558594, 46.47842788696289, -20.807254791259766, -22.909364700317383, 46.073883056640625, 17.478343963623047, -18.981882095336914, -1.2195626497268677, -21.003244400024414, 4.6526198387146, 7.476187229156494, -8.228168487548828, -50.46200180053711, -8.027448654174805, 18.716135025024414, -9.011754989624023, -28.60388946533203, -23.033288955688477, 25.165834426879883, -27.30576515197754, -23.972911834716797, -4.478514671325684, 5.468709945678711, 11.229004859924316, 4.06450891494751, 36.94145202636719, 20.127696990966797, -49.16791915893555, -7.335969924926758, -4.54755163192749, 25.678462982177734, 15.311524391174316, -47.09389877319336, -24.514633178710938, 11.638601303100586, -13.767818450927734, 41.442543029785156, 24.633405685424805, 41.81044006347656, -3.6253607273101807, 51.45603942871094, 44.324012756347656, -6.921177864074707, 15.416303634643555, 53.28693771362305, 10.941845893859863, 4.545501232147217, 15.541898727416992, 19.218664169311523, -14.486043930053711, -37.41566848754883, -20.558935165405273, -22.1485538482666, 36.04938507080078, -58.5737190246582, -72.04011535644531, -34.57915115356445, 35.826133728027344, -17.18362808227539, -20.595935821533203, -39.58805847167969, -22.666650772094727, -29.398523330688477, 11.166180610656738, -13.056154251098633, -15.988885879516602, 22.17369842529297, 53.69690704345703, -41.76435470581055, -34.50335693359375, 14.258679389953613, 7.816666126251221, 6.22515869140625, -5.439151763916016, 3.6006460189819336, 17.899959564208984, -32.80671691894531, 9.011119842529297, -18.74803352355957, -9.145051002502441, -10.938986778259277, -9.459515571594238, 11.261157035827637, 10.521302223205566, -25.10637855529785, -42.28740310668945, 11.891057968139648, 13.14483642578125, 2.8332889080047607, -37.55937576293945, 21.4040584564209, -4.11026668548584, -16.620370864868164, 49.248966217041016, -12.341523170471191, 37.202880859375, -13.924819946289062, -23.090896606445312, 51.93608474731445, 13.271491050720215, -50.576271057128906, -9.188603401184082, -13.622464179992676, -9.133675575256348, 27.56639862060547, -35.104225158691406, -8.765328407287598, 19.003620147705078, -4.548475742340088, -20.291988372802734, -1.1603341102600098, 62.84192657470703, 21.275161743164062, -6.103003978729248, 27.102828979492188, -33.98904800415039, 15.833866119384766, 3.9315788745880127, 31.851980209350586, 54.857418060302734, -3.43269944190979, -2.238574981689453, -14.878697395324707, 16.09589385986328, 35.40923309326172, 17.660953521728516, 41.4123420715332, -10.0606107711792, -59.041988372802734, 47.14851379394531, 8.845915794372559, -13.360260963439941, -77.97540283203125, 29.08689308166504, 12.172501564025879, 8.210602760314941, -7.745059967041016, -6.259040355682373, 16.260679244995117, 23.696062088012695, 41.13404083251953, 21.140867233276367, 37.553253173828125, 25.047834396362305, 30.237972259521484, 16.97745704650879, 22.306922912597656, 48.06803894042969, 34.474327087402344, -8.259539604187012, 0.5049285292625427, -6.67280387878418, 42.21989822387695, 12.085844039916992, -13.13049602508545, -29.025358200073242, -35.234214782714844, 10.142805099487305, -6.683263301849365, 4.160393714904785, -45.94878387451172, -1.1389116048812866, 31.56776237487793, -33.20631408691406, -3.849428176879883, -6.171418190002441, -7.243110179901123, -21.34869384765625, -20.260847091674805, 28.283906936645508, -65.67826080322266, 4.854942798614502, -14.735548973083496, -21.635700225830078, -8.071131706237793, -27.044918060302734, 45.918182373046875, 11.802114486694336, 27.33780288696289, -35.9140739440918, -41.457149505615234, -23.74226188659668, 45.017127990722656, 49.831268310546875, -1.6716513633728027, 9.188210487365723, -53.23237991333008, 7.309929847717285, -24.311538696289062, -16.800146102905273, -35.78044509887695, -7.629575729370117, -41.04188919067383, -3.4616878032684326, 17.5504150390625, -38.279808044433594, 5.874391555786133, -1.4827686548233032, -18.8566951751709, 19.140316009521484, -11.098129272460938, -14.90470027923584, 20.81495475769043, 5.345787048339844, -52.528045654296875, 34.316322326660156, -59.06169509887695, -1.9000763893127441, 44.63291931152344, 48.85165786743164, -20.9328556060791, -3.015038251876831, 24.293420791625977, 21.216386795043945, 8.886487007141113, 47.751075744628906, -36.457733154296875, -20.411319732666016, -45.529685974121094, -29.380191802978516, 6.1241631507873535, 57.41259002685547, 42.93673324584961, -6.5506415367126465, 9.942312240600586, 39.85533905029297, -20.026859283447266, -8.67619800567627, 72.71944427490234, 45.70671844482422, 40.24419021606445, -32.99357223510742, 25.514007568359375, 51.08723068237305, 33.53252029418945, -49.07933044433594, 2.5293781757354736, -48.425411224365234, 25.070764541625977, -30.12111473083496, -23.48436164855957, 1.7151806354522705, -13.800945281982422, 24.249584197998047, -24.748647689819336, -14.628732681274414, 12.115630149841309, -5.22347354888916, -17.75874900817871, -3.4314427375793457, 30.180126190185547, 14.651057243347168, 19.13231658935547, -51.54192352294922, 19.888212203979492, 36.34281539916992, 10.033821105957031, -50.86992645263672, -27.63134002685547, -27.686532974243164, -8.95866870880127, -7.253815174102783, -25.916820526123047, 6.744792461395264, 24.820024490356445, -61.31822204589844, 26.108306884765625, -3.2910280227661133, -43.10447692871094, -59.8013801574707, -30.33847999572754, -2.6192493438720703, -0.8447077870368958, -59.65095520019531, -99.76091003417969, 7.755452632904053, -28.06642723083496, -4.3100457191467285, 8.782438278198242, 3.9246439933776855, 6.288114547729492, -28.31531524658203, 31.444276809692383, 7.723030090332031, 10.177809715270996, 45.90040588378906, -34.26351547241211, -33.8896484375, 39.527469635009766, -39.95582580566406, 26.748165130615234, -2.4318580627441406, -21.121551513671875, 27.291473388671875, 4.183635711669922, 0.06470735371112823, -31.150453567504883, 25.276142120361328, 8.011307716369629, -13.240199089050293, 23.08032989501953, -21.403711318969727, -13.674849510192871, -44.875213623046875, -4.494636535644531, -0.4132402539253235, -5.269925594329834, -30.188573837280273, 10.703574180603027, 32.966793060302734, -72.79051971435547, -34.35136032104492, -3.0905470848083496, 6.418150901794434, -12.32216739654541, 51.24700164794922, 8.220523834228516, -10.550411224365234, -4.02390193939209, 51.28323745727539, -17.26278305053711, 59.8513298034668, 7.934791564941406, 11.59585952758789, 21.014814376831055, -20.93158531188965, -36.764739990234375, 13.734314918518066, 27.852445602416992, -4.731042385101318, 22.129541397094727, 43.10841751098633, 11.291481018066406, -21.59247589111328, -38.13596725463867, 2.5116264820098877, -4.766822338104248, -69.10521697998047, -12.134048461914062, -29.358821868896484, -42.806304931640625, 27.466636657714844, -5.748645305633545, 42.01128005981445, -14.744155883789062, -35.43764877319336, -23.584463119506836, 15.668163299560547, 17.962299346923828, 13.007563591003418, -3.004970073699951, -35.372310638427734, -45.7265625, 18.455387115478516, -47.659767150878906, 46.04459762573242, -22.19261360168457, 33.03627014160156, -22.680856704711914, -53.09059143066406, 15.082462310791016, -16.259431838989258, -30.11836814880371, 23.112730026245117, -6.283164024353027, -36.07954788208008, -5.312990665435791, -55.30236053466797, 4.62693452835083, -67.62638854980469, -17.010623931884766, -22.01286506652832, -9.89288330078125, -45.14243698120117, -51.90579605102539, 29.775901794433594, -64.72113800048828, -32.19007873535156, 3.3155105113983154, -13.757134437561035, 10.563801765441895, -14.533954620361328, -60.34502410888672, -11.377069473266602, 55.5331916809082, -43.85577392578125, -12.766003608703613, -23.211790084838867, 37.567176818847656, -10.003728866577148, 24.88922882080078, -29.94989776611328, -40.10773849487305, -9.520787239074707, -19.766260147094727, 27.260601043701172, -12.280272483825684, 30.67985725402832, -0.3758968710899353, -35.56987762451172, 5.535037517547607, -6.851384162902832, 11.424571990966797, -47.64976501464844, -31.579334259033203, -12.425384521484375, 28.848621368408203, 19.029348373413086, -4.256695747375488, 6.3104705810546875, -45.55007553100586, 6.395063400268555, 19.85024642944336, -11.487421035766602, 2.6331048011779785, 5.428059101104736, 34.50592041015625, 33.998626708984375, -23.087522506713867, 14.524897575378418, 15.632819175720215, 31.62908363342285, -23.65316390991211, -0.17186526954174042, -68.54600524902344, 14.440018653869629, 35.344139099121094, 11.125332832336426, -23.003984451293945, -31.622364044189453, -14.944806098937988, 6.704281806945801, 23.427743911743164, 17.304615020751953, -63.04280471801758, 18.561798095703125, 46.279052734375, -10.978107452392578, 20.573158264160156, 48.481327056884766, -15.335381507873535, 11.87791919708252, 12.902935028076172, -4.43151330947876, -41.5784912109375, -6.757908821105957, 3.120414972305298, -32.47795867919922, -52.47895812988281, -52.90782928466797, 10.800353050231934, 43.725040435791016, -10.975849151611328, -0.5427328944206238, 32.15062713623047, 3.1169700622558594, -17.887300491333008, -24.01121711730957, 41.207557678222656, -52.402183532714844, -28.545028686523438, 37.975528717041016, -32.246456146240234, 37.499229431152344, 56.386474609375, 75.67129516601562, 29.36263084411621, 1.0381803512573242, 13.722805976867676, 51.75975036621094, -37.948760986328125, -10.523205757141113, -70.69573974609375, 27.352712631225586, 9.096840858459473, 13.620325088500977, -37.85392379760742, -15.717989921569824, -18.087793350219727, -29.576190948486328, -6.68054723739624, 22.637537002563477, 50.861122131347656, -2.688917875289917, -20.688007354736328, 33.342838287353516, -35.446205139160156, 39.50221633911133, -9.874212265014648, 88.23804473876953, 19.03803253173828, 13.456080436706543, -20.240922927856445, 2.598186731338501, -36.073665618896484, 37.963592529296875, 26.082483291625977, 27.85811996459961, 16.19890594482422, -62.29973220825195, 54.795257568359375, 21.384553909301758, -2.1059882640838623, -47.653751373291016, -19.513477325439453, -14.903432846069336, -37.824119567871094, -9.875285148620605, -11.015294075012207, 6.407200813293457, -28.296178817749023, 51.923988342285156, -21.657878875732422, -21.956586837768555, -79.80492401123047, 36.02203369140625, -7.124396324157715, 26.194812774658203, -40.01445007324219, 23.277196884155273, 27.911075592041016, -12.557414054870605, -0.8906468152999878, -55.75132751464844, 12.934778213500977, -31.387861251831055, 5.150509357452393, -64.8743667602539, -41.91456985473633, -4.658817768096924, 23.795473098754883, 36.24276351928711, 22.934589385986328, 1.691988229751587, 24.77587890625, 4.389057636260986, 52.4923210144043, -56.472129821777344, 31.177751541137695, 46.510475158691406, 13.4612398147583, 25.192235946655273, -7.831207275390625, 15.285252571105957, -54.45077896118164, -22.323047637939453, -9.20363712310791, 3.7410144805908203, -27.374540328979492, 0.30048227310180664, 17.526371002197266, 53.941314697265625, -29.682649612426758, -40.71871566772461, -34.50807189941406, 5.688929557800293, -10.690235137939453, 1.0635926723480225, -18.486726760864258, -13.710726737976074, 35.3443603515625, -3.9908673763275146, -27.104843139648438, 23.469327926635742, 41.184024810791016, -62.67791748046875, -11.843066215515137, -25.561019897460938, 21.54305076599121, -5.760970115661621, 8.157952308654785, 28.182401657104492, -50.59389114379883, 16.49358367919922, 45.72666931152344, 7.692548751831055, 28.206647872924805, 30.988487243652344, -26.177461624145508, 9.678237915039062, -2.470087766647339, -12.121254920959473, 7.456576347351074, 14.104164123535156, 37.83285903930664, 55.42245864868164, 19.939085006713867, -36.166961669921875, -6.026975154876709, -24.561248779296875, -21.312198638916016, 2.751610517501831, 54.5525016784668, 6.872066020965576, 16.15526580810547, -52.59666061401367, -76.58000946044922, 13.337869644165039, -40.76836013793945, -35.207252502441406, -51.087032318115234, 25.72801971435547, 9.00007152557373, 6.350251197814941, 37.73054885864258, 2.5861129760742188, -13.72197151184082, 17.539371490478516, -13.073732376098633, 24.380369186401367, 35.18014144897461, -10.377285957336426, -38.108680725097656, 13.167896270751953, -30.781824111938477, 35.202030181884766, -5.358907699584961, -40.627655029296875, 32.993595123291016, 25.180681228637695, -14.31055736541748, -9.15685749053955, -11.162251472473145, -22.691558837890625, -26.117664337158203, 22.273975372314453, -23.104658126831055, 2.1243736743927, 9.671340942382812, -31.512264251708984, 7.448183059692383, -3.538029909133911, 48.76922607421875, 28.158369064331055, -16.123992919921875, 38.21437072753906, -35.943687438964844, 1.8929107189178467, 6.6473002433776855, 60.4462776184082, -9.560314178466797, 14.504801750183105, -20.1412410736084, -14.707524299621582, -48.70572280883789, -0.7535948157310486, 4.311753273010254, -3.88529372215271, 31.40749740600586, -0.2265094518661499, 23.187623977661133, 24.734756469726562, 29.206663131713867, -38.001861572265625, 0.592764139175415, -48.82807159423828, -27.069156646728516, -4.899583339691162, 64.7137451171875, 17.584529876708984, -14.272210121154785, 46.65232467651367, 13.21679401397705, 26.402389526367188, -9.667171478271484, 34.581809997558594, -3.8755533695220947, -3.1391148567199707, 0.3229949474334717, 60.089820861816406, 10.33520793914795, -3.2455570697784424, 0.12109025567770004, 5.240285396575928, -30.357534408569336, -25.00637435913086, -21.93033218383789, 41.38678741455078, -15.974194526672363, -45.390907287597656, 33.01565170288086, -30.846107482910156, 22.150346755981445, 8.677480697631836, -26.3750057220459, 22.1292667388916, 20.58001136779785, -4.266605854034424, 9.029821395874023, -29.110689163208008, 19.742345809936523, -11.088452339172363, 46.49267578125, -15.723867416381836, 74.4271240234375, 21.288799285888672, -14.013673782348633, 13.961638450622559, -6.402580738067627, -9.827957153320312, -25.683135986328125, 24.100074768066406, 9.665902137756348, 37.193111419677734, 36.217716217041016, -6.4833502769470215, 41.0960693359375, 18.208345413208008, 30.248268127441406, -14.69533920288086, 39.102317810058594, 2.4578006267547607, 47.75443649291992, -20.957321166992188, -8.855391502380371, 6.421876907348633, -3.2591476440429688, -12.528587341308594, 12.507795333862305, -14.652135848999023, -33.642173767089844, -41.356529235839844, 14.437840461730957, 11.846384048461914, -22.54878044128418, -9.876646041870117, -2.5851757526397705, 10.164545059204102, 8.13228988647461, 52.672691345214844, 1.7455774545669556, 1.1188158988952637, 37.552696228027344, -6.553112030029297, -18.487224578857422, 9.231366157531738, 107.45806121826172, -6.205958843231201, 8.437999725341797, -4.572597503662109, 4.145524024963379, 8.894010543823242, -41.62744903564453, 30.39434242248535, -32.01605987548828, 2.7818663120269775, -48.20944595336914, 32.84444046020508, -13.118307113647461, 46.99712371826172, -23.901254653930664, -9.627971649169922, -31.433883666992188, 19.256235122680664, 16.824565887451172, 17.390132904052734, -34.03525924682617, 7.589156627655029, -34.26695251464844, 6.647037506103516, -8.854301452636719, -21.190919876098633, -9.256570816040039, 23.965768814086914, -20.376842498779297, 12.820205688476562, 12.489130973815918, -18.247770309448242, -40.04536437988281, -4.527960777282715, -14.6856050491333, 38.06822967529297, 10.084757804870605, 21.03788185119629, -22.11220359802246, 42.142635345458984, 59.58601379394531, 10.735459327697754, 12.970211029052734, -9.845488548278809, 3.5336575508117676, -21.153057098388672, -29.863473892211914, -40.82052230834961, -27.208528518676758, 41.51945114135742, -8.528974533081055, -40.0410041809082, 34.83577346801758, 72.02079772949219, -10.972804069519043, -28.267606735229492, 23.240070343017578, 43.42301940917969, -0.7302165031433105, -4.179649829864502, -44.432979583740234, -11.359597206115723, -22.093292236328125, 4.217674732208252, -41.343196868896484, -59.85548782348633, -19.04509162902832, 24.471521377563477, -20.742910385131836, 54.89072799682617, -20.74721908569336, 51.681541442871094, -33.23884963989258, 6.937089920043945, -35.63424301147461, -13.8634033203125, -11.052935600280762, -17.62429428100586, 35.70914840698242, -1.2577966451644897, 14.397018432617188, 35.05248260498047, 60.67643737792969, -47.20673751831055, 76.47411346435547, 24.497282028198242, 10.144932746887207, 24.768321990966797, -63.18758010864258, 27.087493896484375, 51.186092376708984, 44.46416473388672, -25.036937713623047, 35.37483215332031, -37.66023254394531, -22.325618743896484, -1.3533647060394287, -44.018795013427734, -66.93778228759766, 14.355277061462402, -0.8717638254165649, -4.1236042976379395, 15.02615737915039, -8.716924667358398, 3.783297300338745, -31.371868133544922, -9.75425910949707, 51.8502311706543, 22.292524337768555, 39.999778747558594, -38.51496505737305, 2.3107011318206787, -4.460488319396973, 7.0395894050598145, -56.52783203125, -40.16074752807617, 7.25275182723999, 62.88056182861328, -10.750021934509277, 15.252689361572266, 1.321271538734436, -9.381758689880371, 25.083993911743164, 9.558967590332031, 22.768293380737305, -0.21328754723072052, 23.51523208618164, -17.123342514038086, -5.017083168029785, -33.690162658691406, -0.5259042978286743, -13.917360305786133, -5.609324932098389, 41.049949645996094, 2.8716883659362793, -12.00228500366211, 11.89527702331543, 22.59984016418457, 6.703838348388672, 37.618553161621094, -14.693248748779297, -6.944197654724121, -21.671804428100586, -2.0657708644866943, 35.11561584472656, -5.2819905281066895, 76.36749267578125, 7.8796820640563965, -32.47279739379883, 39.72621536254883, 0.48430395126342773, -14.16070556640625, -13.494946479797363, -67.93640899658203, -6.605655193328857, 27.689680099487305, 11.386927604675293, -29.942358016967773, -38.43169403076172, -18.289451599121094, 53.04804992675781, 17.060916900634766, 38.402042388916016, 16.547521591186523, 58.18671417236328, -27.081226348876953, 6.22030782699585, 1.8842965364456177, -23.8983097076416, 20.530078887939453, 23.311336517333984, -17.338584899902344, -5.7874650955200195, 25.692157745361328, 77.34571075439453, 26.754554748535156, -3.4222187995910645, -2.570784568786621, 1.788971185684204, 3.834696054458618, -51.07295227050781, -6.854917526245117, -38.11173629760742, 83.43942260742188, 15.346324920654297, -5.863138675689697, -42.612640380859375, -0.16629688441753387, -28.740922927856445, 40.820308685302734, 12.815579414367676, 67.92293548583984, 14.465375900268555, 22.430631637573242, 37.68046569824219, 10.814775466918945, -11.090977668762207, 16.697935104370117, 41.896854400634766, 25.565513610839844, 12.128652572631836, 15.82470989227295, -7.140771865844727, -12.237299919128418, -16.749637603759766, -19.654695510864258, 8.379598617553711, 34.57888412475586, 58.795982360839844, -16.502941131591797, 4.039751052856445, 9.230273246765137, 25.113391876220703, -14.775522232055664, -22.91132354736328, -34.66780471801758, 2.723703622817993, 27.439058303833008, 35.160396575927734, 4.2760910987854, 3.2639944553375244, -26.217348098754883, -48.30232238769531, -4.822243690490723, 8.216413497924805, 3.7160449028015137, -12.842132568359375, -48.05014419555664, -14.6964111328125, -58.611000061035156, 27.60686683654785, 14.49825382232666, -17.09947395324707, 13.656534194946289, 35.832454681396484, 25.029672622680664, -0.03716157004237175, 43.75870895385742, 8.817861557006836, 6.804016590118408, 15.462112426757812, -15.621612548828125, 36.75577926635742, -39.0032958984375, 15.335638999938965, -31.258947372436523, 6.702671051025391, -49.669227600097656, 41.35633087158203, 18.404434204101562, 9.790032386779785, 33.42610168457031, -16.143795013427734, -35.37130355834961, -28.75490379333496, -2.535435438156128, -2.1784002780914307, -23.73309898376465, -42.75832748413086, -5.778995513916016, 13.996288299560547, -5.123714447021484, 46.79041290283203, -28.098533630371094, 3.949139356613159, 35.8760986328125, -3.0732216835021973, -10.842941284179688, -31.222915649414062, -35.33799743652344, 69.55799865722656, 11.316338539123535, -60.040740966796875, -11.110479354858398, 8.422202110290527, 24.789928436279297, 61.038814544677734, 18.114421844482422, 8.685784339904785, 46.606258392333984, 49.69511032104492, 10.100997924804688, 28.30910873413086, 7.963501930236816, -42.461185455322266, -14.827157974243164, 39.610862731933594, 19.652568817138672, 55.325225830078125, -10.818888664245605, -19.9547119140625, 31.476566314697266, 64.990478515625, -5.934523582458496, 68.98311614990234, 8.727701187133789, 40.1334228515625, 6.143066883087158, 2.134490728378296, -36.02853012084961, 30.545618057250977, 28.020112991333008, -4.9725446701049805, 25.619274139404297, 17.9227294921875, -34.31704330444336, -18.36962127685547, -1.5124307870864868, 62.77143096923828, 8.276854515075684, -2.5190746784210205, 32.720664978027344, 36.043418884277344, -22.864530563354492, 13.507749557495117, -45.45466613769531, -41.043373107910156, 32.75924301147461, 17.613004684448242, 15.498283386230469, -16.155630111694336, -9.751486778259277, -48.323360443115234, -24.436845779418945, 2.0315513610839844, 28.305458068847656, 7.6581807136535645, 62.86050033569336, 12.686019897460938, -21.498371124267578, 61.96889877319336, -25.059547424316406, -2.8586907386779785, 39.40930938720703, -26.28934097290039, -40.74625015258789, 1.1937001943588257, -37.379119873046875, -44.432579040527344, 48.3363151550293, 38.15325927734375, -7.491448879241943, -13.586763381958008, -5.488123416900635, 9.565853118896484, 88.09700775146484, 19.71981430053711, -21.886289596557617, -23.273719787597656, 29.4312744140625, 55.45127868652344, 18.122913360595703, -49.18960952758789, -37.79634475708008, -5.513827800750732, -13.458582878112793, -30.788043975830078, 62.094512939453125, 15.399556159973145, 41.12661361694336, -47.75032424926758, 26.12242889404297, -3.6780548095703125, 50.98360061645508, -11.759940147399902, 15.143460273742676, 13.693565368652344, -20.509246826171875, 24.060535430908203, -9.140562057495117, -3.549696683883667, -33.81938934326172, 18.474781036376953, -2.707212209701538, -39.55375289916992, -29.417306900024414, 0.3893369138240814, -46.28646469116211, -17.647375106811523, 42.969627380371094, -30.850950241088867, -11.048633575439453, 74.52197265625, 7.496705055236816, 13.231039047241211, 17.312929153442383, -1.5718718767166138, 16.697277069091797, 102.45327758789062, 28.026588439941406, 25.871501922607422, -12.232028007507324, -28.865442276000977, 12.094667434692383, -15.58649730682373, 40.48405838012695, -12.60877799987793, 12.1400146484375, -3.0600836277008057, -10.375568389892578, -8.698956489562988, 33.01601028442383, 21.555625915527344, -43.74728012084961, -14.531045913696289, -22.209815979003906, 23.097089767456055, -15.277032852172852, -18.621469497680664, -22.713346481323242, 42.55473709106445, 27.529441833496094, -59.959468841552734, 0.04733781889081001, -2.1911768913269043, 15.908011436462402, 14.346139907836914, 27.63050651550293, 65.70242309570312, -51.75617599487305, 49.90446472167969, -13.808509826660156, -42.460838317871094, -12.07481861114502, -28.96891212463379, 75.07707214355469, 20.73565101623535, 23.475872039794922, -8.408843040466309, 4.161010265350342, 27.157299041748047, 15.459199905395508, 19.264652252197266, 22.42475700378418, 26.487367630004883, -20.150901794433594, 37.14616012573242, -1.0735712051391602, -14.606466293334961, -28.826080322265625, 28.805355072021484, 14.853331565856934, 46.766082763671875, 4.498505592346191, -14.427446365356445, 12.50192928314209, -4.699559688568115, -15.62289810180664, -45.98390579223633, -31.12122344970703, 25.04126739501953, 13.862344741821289, -26.671472549438477, 12.11414909362793, 40.70478439331055, -7.761178493499756, 24.362098693847656, 12.129600524902344, 15.00758171081543, 21.502639770507812, -36.23467254638672, 22.878847122192383, 16.26201057434082, 60.05975341796875, -3.4110114574432373, 40.93265151977539, 19.107864379882812, -20.15127182006836, -48.05457305908203, -25.995502471923828, -8.121371269226074, -8.770439147949219, 14.86346435546875, 18.307209014892578, -12.1112642288208, -16.396883010864258, -30.68903160095215, 80.63844299316406, 47.23545837402344, -40.2819938659668, 15.93702220916748, -8.922953605651855, 1.3588457107543945, 58.62749481201172, -26.640243530273438, 4.636876583099365, -5.737908363342285, 31.599748611450195, 44.604000091552734, -8.764827728271484, -23.823938369750977, -7.591061115264893, -14.798243522644043, -59.322486877441406, -32.800296783447266, -31.679664611816406, -39.857460021972656, -36.96257781982422, -56.884525299072266, -12.575103759765625, -25.263784408569336, 1.4816592931747437, -31.379413604736328, 25.233234405517578, -24.93754005432129, 25.043277740478516, -2.475487232208252, 2.9555110931396484, -27.68792724609375, -22.322864532470703, 3.6108293533325195, -31.155975341796875, -18.876270294189453, 16.999792098999023, 7.474749565124512, -22.65018653869629, 29.048948287963867, -38.7289924621582, 20.311790466308594, 4.182169437408447, 8.999469757080078, -12.803922653198242, 7.412388324737549, 21.339920043945312, -8.075981140136719, 21.285451889038086, -0.5277746319770813, 8.686051368713379, 32.537166595458984, -16.64752197265625, 4.751948356628418, 12.410173416137695, 15.560419082641602, 44.986385345458984, 1.6075656414031982, -3.966832399368286, 78.27195739746094, -28.700103759765625, -55.1888427734375, -12.758261680603027, 22.969234466552734, -9.496399879455566, 58.777183532714844, 14.37894058227539, 7.733435153961182, 44.57110595703125, 8.553157806396484, -14.196744918823242, -2.791630268096924, 12.342105865478516, 8.261043548583984, 15.557862281799316, -14.912190437316895, 24.186752319335938, -36.75568389892578, 8.091353416442871, 72.71915435791016, 32.40238571166992, 20.840208053588867, -25.352441787719727, -42.76182556152344, 60.719482421875, 22.57460594177246, 20.272268295288086, 19.595325469970703, -9.008773803710938, 45.98198318481445, -41.20475769042969, -36.45219039916992, -3.8574447631835938, 8.456536293029785, 16.990406036376953, -12.642061233520508, 27.247106552124023, -31.606203079223633, 6.150293350219727, 38.60547637939453, 2.2893073558807373, 34.72685623168945, 65.38716888427734, 30.17671775817871, 41.81227111816406, 67.9487075805664, 9.795166015625, 13.8494234085083, 37.952449798583984, 12.24051570892334, -36.064815521240234, -67.05171203613281, 4.630650520324707, -29.575458526611328, -11.45151424407959, 16.31126594543457, -35.51980209350586, 19.06603240966797, 9.704493522644043, -50.06439971923828, -12.701144218444824, -15.390280723571777, 10.101985931396484, -16.068952560424805, -7.725800037384033, -15.256889343261719, -13.670254707336426, 51.78263473510742, -12.726515769958496, 45.98508834838867, 33.888389587402344, -31.401865005493164, -55.015525817871094, -37.40458679199219, -27.81771469116211, -4.011357307434082, 49.70165252685547, 13.988776206970215, 20.32526397705078, 28.37166976928711, 0.11952283978462219, -2.906255006790161, -4.448493957519531, 27.21184539794922, -14.893290519714355, 10.530131340026855, -38.38148880004883, 54.80576705932617, -3.0230133533477783, 1.2231703996658325, -56.51716232299805, 21.01813316345215, 15.356420516967773, 95.9191665649414, 22.967559814453125, -9.796886444091797, -30.128767013549805, -43.537418365478516, 42.634178161621094, 67.0266342163086, -23.168718338012695, 0.3113989531993866, -57.12668228149414, 8.308097839355469, 44.08856201171875, 64.75645446777344, -53.360511779785156, 6.577109336853027, -8.318355560302734, 21.343339920043945, 6.706603527069092, -23.096046447753906, -1.7220754623413086, 40.40385055541992, -95.05786895751953, -41.412132263183594, 39.802154541015625, -3.3283944129943848, -60.07329177856445, 33.268672943115234, 11.480997085571289, 0.8263357281684875, 39.41085433959961, -31.86716079711914, 65.11846923828125, 40.28001022338867, 72.57621002197266, 6.596749305725098, -29.885604858398438, -13.203191757202148, -45.798213958740234, 25.656343460083008, 29.094985961914062, -1.4554718732833862, 2.4817168712615967, -2.983229398727417, 2.0557286739349365, -14.659757614135742, 59.27907180786133, -69.62179565429688, -30.343473434448242, 42.26021194458008, 33.7877197265625, -22.49420166015625, 2.8012473583221436, -9.288349151611328, 7.416054725646973, -20.61313819885254, -36.24639129638672, -6.796678066253662, 20.79732322692871, 8.277010917663574, -31.2446231842041, 19.29347801208496, -17.035350799560547, -10.598917007446289, -68.31258392333984, -13.061896324157715, 30.959705352783203, -29.442949295043945, -20.853351593017578, 9.94792652130127, 36.807960510253906, 38.586952209472656, -1.6328434944152832, -28.06311798095703, -24.93059730529785, -17.0208740234375, -6.532692909240723, -3.006685256958008, 62.10139846801758, 28.487882614135742, -62.08597183227539, 27.175846099853516, 3.98421049118042, 31.585018157958984, 37.759586334228516, -19.545608520507812, 43.37525939941406, -2.6459341049194336, 54.396244049072266, 14.841644287109375, 6.44556999206543, -20.11922264099121, 3.61688232421875, 9.468436241149902, 16.319021224975586, -2.6833555698394775, -34.70515441894531, -27.037837982177734, -9.176856994628906, 25.27528953552246, 20.707355499267578, -25.726530075073242, -39.96846008300781, -24.009857177734375, 16.055418014526367, -1.59173583984375, 14.059279441833496, 8.929088592529297, 8.032599449157715, 52.09602355957031, 7.502829551696777, -41.542633056640625, 29.356800079345703, -47.617557525634766, -13.975420951843262, -60.95909881591797, -2.5587844848632812, -59.34685516357422, -42.4415168762207, -21.97605323791504, 4.68618106842041, 20.288944244384766, -59.625762939453125, 26.84931755065918, -17.620271682739258, -8.447319984436035, -2.56140398979187, -40.32130813598633, 39.11845397949219, -75.79856872558594, -45.61766052246094, 9.119424819946289, 18.2166690826416, 8.033650398254395, -0.12012780457735062, -25.019792556762695, -0.6914112567901611, -12.227540016174316, -53.21124267578125, 31.394535064697266, -5.758097171783447, 34.256019592285156, 15.27208137512207, -27.225250244140625, -43.34137725830078, 21.257728576660156, 18.51633071899414, -19.273679733276367, -19.943782806396484, -29.679901123046875, -15.40185546875, 52.69572830200195, -12.464092254638672, 40.476806640625, 22.296972274780273, -29.182966232299805, -14.739524841308594, -59.796424865722656, -17.685510635375977, 42.17354202270508, 26.54813003540039, 8.460441589355469, -19.486093521118164, 23.517545700073242, -45.45689392089844, 1.7613424062728882, 13.879446029663086, 22.230194091796875, 2.761976957321167, -11.802545547485352, 41.28211975097656, -19.225004196166992, 23.83219337463379, 6.497369289398193, 4.561530590057373, 62.63456726074219, -3.4557406902313232, 3.9565176963806152, 26.842891693115234, 22.26138687133789, -29.922588348388672, 52.877685546875, 10.877323150634766, 6.881720066070557, -7.819457530975342, 30.096670150756836, -60.58229064941406, -10.1210355758667, 10.41775894165039, 5.131484508514404, -82.64993286132812, 60.30964279174805, 19.462923049926758, -8.703681945800781, 8.572006225585938, -15.66009521484375, 17.293725967407227, -27.021648406982422, -16.910188674926758, -19.11026954650879, -17.32843017578125, -11.725542068481445, 17.401430130004883, 28.394250869750977, 43.41386795043945, -1.8402111530303955, 11.102530479431152, 24.21014404296875, -40.458438873291016, -30.78432846069336, 11.414867401123047, 14.531981468200684, -18.494245529174805, -52.2664909362793, 3.9181888103485107, -32.4615364074707, 13.025824546813965, -25.58941078186035, 37.295570373535156, -88.63115692138672, -31.433494567871094, 26.369403839111328, -1.7410709857940674, 6.797760963439941, 32.629451751708984, -16.22495460510254, 37.27113723754883, 12.859298706054688, -19.71267318725586, -24.246158599853516, -2.4577393531799316, 16.801502227783203, -56.87068176269531, 15.177367210388184, 13.108784675598145, 44.40637969970703, -30.046430587768555, 20.5225830078125, 38.20478439331055, -0.9043735861778259, 48.936973571777344, -2.6907782554626465, 5.827202320098877, -32.05863571166992, 22.163028717041016, -32.488037109375, 47.92367172241211, 8.276527404785156, 15.088262557983398, 3.8345723152160645, -39.531654357910156, 4.494979381561279, 50.7355842590332, -21.405887603759766, 15.8703031539917, -54.427040100097656, 11.340210914611816, 13.940412521362305, -0.3530120253562927, -29.275043487548828, -31.666213989257812, -33.10882568359375, 14.027714729309082, 19.81313133239746, -34.32557678222656, 16.457548141479492, 23.69795799255371, 4.87477445602417, -25.392595291137695, -2.5726027488708496, 0.6577984094619751, -35.367305755615234, -24.180879592895508, 9.927757263183594, 8.765454292297363, -11.52657699584961, 12.293612480163574, 21.207595825195312, -44.174232482910156, 55.37770080566406, -30.629981994628906, -31.021936416625977, -24.354883193969727, 9.66952133178711, -4.872913837432861, 2.2294259071350098, -7.722173690795898, -23.816524505615234, 76.1734619140625, -32.264060974121094, -13.457143783569336, -22.36321449279785, -28.537960052490234, 50.104373931884766, -12.06192398071289, 28.36744499206543, -21.6113338470459, -29.896499633789062, -32.511356353759766, -28.920635223388672, 2.6986796855926514, -16.82958984375, 3.3968193531036377, -45.75164031982422, 58.93406677246094, -5.322760581970215, 7.099977493286133, 37.82188034057617, -11.105449676513672, 19.856525421142578, 13.688553810119629, -10.288171768188477, 30.35405921936035, -61.065486907958984, -69.79991912841797, -29.0902099609375, -49.13782501220703, 36.70726013183594, -28.66767120361328, 8.368670463562012, -42.996952056884766, -49.98002624511719, -27.590234756469727, -40.086673736572266, -41.2070426940918, 21.456193923950195, -37.117340087890625, 51.520896911621094, 0.5282407402992249, 28.81730842590332, 50.797733306884766, 12.893815994262695, -3.062591075897217, -24.117713928222656, 13.705522537231445, 39.88896942138672, -37.006797790527344, -72.38188171386719, -25.48775863647461, -39.30672073364258, 0.587320864200592, -4.801308631896973, 2.811448574066162, 30.84354019165039, -19.187406539916992, -43.33258819580078, 10.581088066101074, 42.002193450927734, 16.08879852294922, 37.893489837646484, 12.89756965637207, -7.0566840171813965, 5.312952041625977, -46.14240264892578, 11.868490219116211, 8.581579208374023, -32.81520462036133, 22.551177978515625, -20.825408935546875, 5.875302791595459, -1.9903736114501953, -1.2813607454299927, 24.106592178344727, -26.370616912841797, -13.459548950195312, -39.44026184082031, -33.23371505737305, 10.518190383911133, -20.843294143676758, 33.15756607055664, -26.629974365234375, -13.622873306274414, -38.78132629394531, 35.39541244506836, 31.897045135498047, -7.010715007781982, -34.655670166015625, -8.963482856750488, -55.352455139160156, 22.586814880371094, -15.662461280822754, -78.6135025024414, 20.595458984375, 3.721794605255127, -14.588438034057617, -7.4531426429748535, -7.338177680969238, 41.6491584777832, 3.6737189292907715, -36.086456298828125, -32.10565948486328, 58.37950897216797, -4.663514137268066, 10.862898826599121, -32.31993865966797, 25.223556518554688, 31.230579376220703, -26.196300506591797, 20.63962173461914, 38.67230224609375, 24.937274932861328, -24.217405319213867, -20.347896575927734, -19.942020416259766, -50.90653610229492, 28.775157928466797, -5.110324382781982, -16.117597579956055, -63.1169548034668, 18.677698135375977, -12.895586013793945, 5.735929012298584, -16.82083511352539, -34.654624938964844, 10.173038482666016, -41.21763229370117, -14.975872993469238, 5.211545467376709, -38.54890441894531, 29.55379295349121, -1.6697200536727905, -3.3511714935302734, 3.5185627937316895, -8.142958641052246, 9.70560359954834, 13.211390495300293, 35.80512619018555, 13.659443855285645, -40.44419479370117, -14.172955513000488, -22.65338706970215, -31.674320220947266, 27.192277908325195, 54.119415283203125, 7.824646472930908, -17.06589126586914, -18.225082397460938, -5.544831275939941, 1.8931626081466675, -24.53770637512207, -0.9224684834480286, -12.939770698547363, 32.68439483642578, -15.475968360900879, 30.86457633972168, 15.102747917175293, -8.254683494567871, 13.549443244934082, 10.299789428710938, -67.12539672851562, -31.79474449157715, 4.926833629608154, -43.557376861572266, 26.282052993774414, -0.5817866921424866, 24.016971588134766, -6.310177326202393, -6.659183502197266, -17.866012573242188, -27.381072998046875, -13.087759971618652, 50.426109313964844, 45.48533630371094, -15.866754531860352, 6.840846538543701, 40.70246887207031, -14.540895462036133, 34.111656188964844, 35.678157806396484, -19.402076721191406, -20.857519149780273, -2.0684726238250732, -46.798587799072266, -41.86166000366211, -9.539658546447754, -35.11143493652344, 1.235663652420044, 30.4158992767334, 29.159446716308594, 38.88704299926758, 15.002823829650879, 10.814058303833008, 25.44369125366211, -50.432655334472656, 5.237228870391846, 23.446863174438477, -10.91031551361084, 2.6375789642333984, 70.49134063720703, -13.4920015335083, 21.244672775268555, -4.761794090270996, 78.93966674804688, -19.4123592376709, -35.074607849121094, -22.27214241027832, -18.279050827026367, 13.588479042053223, 17.868379592895508, 6.236602783203125, -39.222408294677734, 12.203401565551758, 4.104284286499023, -30.13817596435547, -11.888009071350098, -7.909527778625488, -30.822242736816406, -16.15004539489746, 0.9533868432044983, 16.70393180847168, -2.1584315299987793, -24.523164749145508, -37.02348709106445, 31.513391494750977, 70.14969635009766, -28.197174072265625, -42.701995849609375, -25.08363914489746, -63.67163848876953, -36.5821533203125, -11.23835563659668, -39.23073959350586, 25.117107391357422, 0.5269124507904053, -35.93743896484375, 4.0476765632629395, 31.120391845703125, 1.8852328062057495, -12.094918251037598, 23.685537338256836, -6.4804158210754395, -38.29533386230469, 8.944587707519531, 15.044190406799316, 28.292062759399414, -31.354772567749023, -44.912357330322266, 18.83407211303711, 19.30965805053711, -21.227828979492188, -8.393607139587402, -37.45412063598633, 7.2775444984436035, 15.957437515258789, 99.21205139160156, -5.535769939422607, -2.35221004486084, 9.759171485900879, -42.19744110107422, -33.36946105957031, 33.35733413696289, 27.419857025146484, 3.625737190246582, -17.107255935668945, 10.94546890258789, 55.60581970214844, -20.520950317382812, 42.354652404785156, -11.062267303466797, -44.86582565307617, 4.236724853515625, 13.378365516662598, 21.349365234375, -26.177125930786133, 12.020602226257324, -14.930098533630371, -23.154399871826172, 21.00366973876953, 51.68482971191406, -33.509056091308594, 20.642393112182617, 3.2961666584014893, -11.102540016174316, -25.185386657714844, 19.23969078063965, -74.33489990234375, 29.65559196472168, -3.628277540206909, 3.9781501293182373, 12.803122520446777, -49.404266357421875, -18.665109634399414, -32.6702995300293, -5.1363325119018555, -76.22486114501953, 14.222953796386719, 39.5578498840332, -1.6927765607833862, 24.604124069213867, 28.000789642333984, -11.242339134216309, 33.356300354003906, 29.262420654296875, -13.674887657165527, -23.968730926513672, 6.2219014167785645, -20.86621856689453, 12.265633583068848, -46.69846725463867, -48.34315490722656, -10.155386924743652, 39.71082305908203, -1.09169340133667, -18.387065887451172, 72.94679260253906, 105.86488342285156, 29.155193328857422, 21.392711639404297, -9.533024787902832, 40.80474090576172, -43.76433563232422, -31.581985473632812, -15.824237823486328, 38.75984191894531, -16.097822189331055, -16.3065128326416, 22.45909881591797, -15.985947608947754, -24.403322219848633, -20.083148956298828, -24.095413208007812, 15.364222526550293, -14.4699068069458, 29.816919326782227, 9.401135444641113, -52.66769790649414, -29.303451538085938, -16.814132690429688, 4.3662309646606445, -15.729101181030273, 10.601991653442383, -12.5759859085083, 3.927969455718994, 0.39138883352279663, 14.922804832458496, 10.130346298217773, -14.402931213378906, 20.77315902709961, 30.120885848999023, -16.4005069732666, 41.59775161743164, 66.28571319580078, -9.474583625793457, -24.03476905822754, -8.338025093078613, 19.59816551208496, 26.42402458190918, 58.661712646484375, 19.518224716186523, -3.658545732498169, 4.084224224090576, 33.463809967041016, 2.793198585510254, -0.2967036962509155, 34.739776611328125, -48.266990661621094, -22.254121780395508, -12.5408353805542, 19.478391647338867, 6.522728443145752, -25.808870315551758, 72.54888153076172, 35.958187103271484, 46.126426696777344, -14.495207786560059, 6.638686656951904, 18.855918884277344, -45.819732666015625, -19.34158706665039, -16.075408935546875, 8.773656845092773, -53.925662994384766, -8.9288969039917, 22.010934829711914, 10.518072128295898, 0.7759262919425964, -7.270351886749268, -0.4058471620082855, -38.95802688598633, 43.045745849609375, -10.09018611907959, -9.827048301696777, 32.11203384399414, 1.566552758216858, -25.844186782836914, -11.811902046203613, 48.51673889160156, 59.24446487426758, 26.733617782592773, 11.601061820983887, 10.297941207885742, -20.285240173339844, 39.00630569458008, -45.97183609008789, -56.00873947143555, 4.609433174133301, -56.708438873291016, 36.62458801269531, -47.1989860534668, -15.963300704956055, 17.5098819732666, -4.441839218139648, -8.41762924194336, 38.42313003540039, -47.79359436035156, 4.298635482788086, 30.62615966796875, 13.49463939666748, -26.030187606811523, 1.1595697402954102, 10.080891609191895, 5.710177421569824, -1.787134051322937, 37.8174934387207, 50.68918228149414, 3.2998485565185547, -12.716082572937012, -47.44113540649414, -45.38646697998047, 18.141855239868164, -44.40296936035156, 55.29650115966797, -2.1200437545776367, 18.647462844848633, 39.12486267089844, 13.380426406860352, -53.50099182128906, -38.30724334716797, -5.506657123565674, -27.315528869628906, 39.1873779296875, 34.636287689208984, 8.351156234741211, 14.49713134765625, 29.23009490966797, -14.154170036315918, 14.915002822875977, 7.65880823135376, 47.38795471191406, -22.14513397216797, -43.98332214355469, -1.1878480911254883, 64.0030746459961, 27.84368324279785, 25.952075958251953, 27.311403274536133, -30.966886520385742, -38.703487396240234, 26.235300064086914, -13.033312797546387, 28.133487701416016, -67.39616394042969, 7.324474811553955, -27.152860641479492, -5.398584365844727, -16.42288589477539, 33.53105163574219, 14.799206733703613, -21.893301010131836, 2.6956167221069336, -26.1838321685791, 55.09685134887695, 56.303646087646484, 29.900108337402344, -18.780254364013672, 15.36423110961914, -40.81059265136719, -29.94403076171875, -28.119674682617188, -3.1452479362487793, -52.66606140136719, -36.00632858276367, 7.605690002441406, 3.3203492164611816, -5.98606014251709, 16.33906364440918, -38.68651580810547, -26.33946418762207, -9.012574195861816, -11.479975700378418, 4.667618751525879, -28.31520652770996, -30.20989990234375, -31.69133758544922, 38.09526824951172, 56.3615837097168, -1.2046891450881958, -10.809120178222656, -6.350212574005127, 8.539889335632324, 28.78215980529785, 6.85227108001709, -25.15677833557129, 68.42141723632812, -36.28112030029297, -7.821966171264648, 21.803577423095703, 41.237548828125, -3.730902671813965, -15.475332260131836, -45.64027404785156, 11.429167747497559, 16.09009552001953, 35.99534606933594, 17.80498504638672, -5.6335296630859375, -7.226241111755371, 47.6634521484375, 30.32590675354004, -15.775169372558594, -17.357559204101562, -7.976181507110596, 13.457695960998535, 31.325605392456055, 0.19579154253005981, 26.516393661499023, 2.2698333263397217, 7.549121856689453, 57.029197692871094, -31.30066680908203, -13.416803359985352, 20.96722412109375, -39.078617095947266, -81.20682525634766, -12.723791122436523, 11.340388298034668, -30.407560348510742, -13.593473434448242, 7.629312038421631, -0.06788194179534912, 34.54013442993164, 26.057985305786133, 1.0489184856414795, 35.72277069091797, -7.456410884857178, -0.11603550612926483, -8.624731063842773, 6.0902791023254395, 12.363158226013184, -59.45583724975586, 29.517118453979492, -45.532981872558594, -13.142670631408691, -21.92841339111328, -36.0594482421875, -39.175445556640625, 5.366767406463623, -35.40026092529297, 56.10695266723633, 37.45547103881836, 2.187734842300415, -31.95118522644043, -2.6156530380249023, 12.188045501708984, 7.475580215454102, 44.71736145019531, 52.54266357421875, -4.785233020782471, -13.926156997680664, 11.390975952148438, -7.720542907714844, -23.668609619140625, -33.45962142944336, 21.363357543945312, 16.014896392822266, -22.52392578125, 52.475135803222656, -42.125389099121094, -40.08767318725586, -2.6073966026306152, 24.717021942138672, 22.800722122192383, 32.76274871826172, 7.647670745849609, -28.22603416442871, -1.1169692277908325, 5.5376434326171875, 4.187977313995361, -29.421247482299805, 18.314226150512695, -42.66301727294922, 26.06057357788086, 42.68464279174805, 43.486900329589844, -60.5850944519043, 42.82814025878906, 12.209348678588867, 29.213747024536133, 28.247610092163086, -28.725248336791992, -44.19374465942383, -25.8983154296875, -10.029210090637207, 53.5682258605957, -0.23958681523799896, -3.4769914150238037, 30.799970626831055, 12.882561683654785, 51.32878494262695, 33.24958038330078, 60.89421081542969, 35.420169830322266, -23.788591384887695, 39.334407806396484, -22.647634506225586, 21.258302688598633, -4.117142677307129, 19.23854637145996, 5.371708393096924, -15.53022289276123, 32.40312957763672, -23.260753631591797, 13.105222702026367, 5.358476161956787, 3.789663553237915, 31.68052864074707, -0.5427777767181396, -12.654088020324707, 0.38213616609573364, 38.957298278808594, -30.523406982421875, 17.294443130493164, -15.3583984375, 49.79233932495117, 6.030198097229004, -10.92009449005127, -18.096946716308594, 38.002967834472656, -23.26998519897461, -1.7686219215393066, -35.04299545288086, 26.334461212158203, 22.17560577392578, -23.60835075378418, -30.391212463378906, 2.50706148147583, -22.85108757019043, -0.920153796672821, -55.99356460571289, -16.505680084228516, 30.74811553955078, -12.22690486907959, 9.823543548583984, 12.507281303405762, -54.0299072265625, 2.3811025619506836, 20.852508544921875, 25.246633529663086, -32.870704650878906, 29.078651428222656, 25.83066749572754, -29.751201629638672, -28.521238327026367, 7.491064548492432, 40.99979019165039, 44.05537796020508, -50.033714294433594, -53.16436004638672, 58.76485824584961, -20.55276107788086, 43.728660583496094, -38.80475997924805, 17.844863891601562, -1.2200127840042114, -22.160017013549805, 3.500617504119873, -4.25191593170166, -40.83216094970703, -13.162446975708008, -35.216060638427734, -11.34280014038086, 3.8286428451538086, 0.9238215684890747, -25.883787155151367, -31.170991897583008, 22.97452735900879, -10.6255464553833, 18.45792007446289, -7.059895038604736, -17.955402374267578, 33.99909973144531, -21.554624557495117, -52.85841369628906, 4.109011650085449, 0.831196665763855, -14.549955368041992, -3.8601999282836914, 10.177431106567383, -37.45514678955078, -3.3438990116119385, -17.793542861938477, -9.21080207824707, 18.069690704345703, 26.461666107177734, -14.229726791381836, -17.556039810180664, -12.747437477111816, -20.979307174682617, 20.42317008972168, -15.874463081359863, 12.931903839111328, 20.68486785888672, 50.39396286010742, 5.060315132141113, -42.67456817626953, 4.103177547454834, -9.68416690826416, -10.897910118103027, 15.30156421661377, 2.432342529296875, 0.09291429817676544, 3.6104705333709717, 5.592287540435791, -52.51388931274414, 15.292496681213379, -64.56168365478516, -2.6026206016540527, 27.24587059020996, -44.88884353637695, 6.931361198425293, 20.471677780151367, -3.4518275260925293, 3.2606186866760254, -56.26289367675781, -16.227127075195312, 25.740558624267578, -22.266132354736328, -19.132373809814453, 28.367185592651367, -0.6174484491348267, 30.947702407836914, -18.66893196105957, -28.67306900024414, -11.11294937133789, 28.844528198242188, 28.190095901489258, 9.3423490524292, 6.505002975463867, -1.2530618906021118, -13.788630485534668, -7.6270365715026855, -19.452878952026367, 3.5549252033233643, -12.597034454345703, 30.87245750427246, 51.56435775756836, 38.53102493286133, -21.083181381225586, 6.3657073974609375, 18.53113555908203, 7.599303245544434, 1.2028108835220337, -7.753520488739014, 9.434696197509766, -39.39641571044922, 54.80708312988281, -10.97542667388916, 6.582360744476318, -26.758010864257812, 0.24339789152145386, -35.14817810058594, -27.43386459350586, 12.07253646850586, 42.76438522338867, -59.13301467895508, 16.270923614501953, 29.019514083862305, -4.442065238952637, -9.341907501220703, 11.021418571472168, -65.80817413330078, -35.252803802490234, -18.689788818359375, -18.431941986083984, -9.060585975646973, 58.951255798339844, 12.188203811645508, 9.367781639099121, 6.960663318634033, -23.569177627563477, -21.801687240600586, 7.4481048583984375, -19.19253921508789, -6.171666622161865, 17.863452911376953, 30.149738311767578, 18.910856246948242, -4.663313865661621, -54.33653259277344, -2.5293099880218506, -18.3151798248291, 41.46143341064453, -30.082929611206055, -11.46345043182373, -3.805798053741455, -7.854305267333984, 21.206504821777344, -8.245659828186035, -41.92100524902344, 11.999350547790527, -31.96962547302246, 63.06061553955078, 6.730985641479492, 27.927139282226562, 27.29991340637207, -14.045272827148438, -42.573421478271484, -17.39871597290039, -28.10870361328125, 56.21564865112305, 15.07443618774414, -23.092618942260742, 49.42626190185547, -8.645438194274902, 35.860679626464844, 45.79890060424805, -13.163131713867188, -16.158924102783203, 11.443563461303711, -1.1176955699920654, -62.734214782714844, -9.727059364318848, 7.67213249206543, 4.20769739151001, -38.0274772644043, -13.229605674743652, -24.200376510620117, -14.334247589111328, 46.10463333129883, -26.984533309936523, 28.814390182495117, -5.738094806671143, -54.76622009277344, 35.297393798828125, 14.389608383178711, -31.84444808959961, 85.69477844238281, -13.86921501159668, 4.888635158538818, -29.449949264526367, -47.602783203125, -51.27513122558594, 24.70047378540039, -34.90463638305664, -86.84681701660156, -1.466658115386963, -43.55753707885742, 63.893043518066406, -1.956945538520813, -37.41450881958008, 9.315155982971191, 15.440847396850586, 5.95145320892334, 1.8691848516464233, -6.427055835723877, -24.556730270385742, 48.3865852355957, 2.586594343185425, 20.884828567504883, 31.15403938293457, 9.489027976989746, 43.9849739074707, 29.17540168762207, 10.195168495178223, -29.80577278137207, 0.5273501873016357, -27.840129852294922, 7.699765682220459, 0.18870264291763306, 23.79566192626953, -7.9090895652771, -22.005563735961914, 2.5839149951934814, 7.499545574188232, -13.492091178894043, -2.9158873558044434, 2.047473430633545, 21.127962112426758, 6.046865463256836, 13.331941604614258, -20.52587890625, 48.02539825439453, 11.403292655944824, -42.897830963134766, -30.474679946899414, -7.9357008934021, 63.25190353393555, -69.93201446533203, 28.99300193786621, 14.931239128112793, 34.003292083740234, -36.32765197753906, -34.54541015625, -27.20601463317871, 22.497194290161133, -51.81109619140625, 9.059895515441895, 30.736499786376953, 26.190122604370117, -27.994640350341797, -18.243240356445312, -24.04698944091797, -25.362796783447266, -7.791659832000732, -1.5753657817840576, 24.972381591796875, 33.22244644165039, 12.76024055480957, 54.469669342041016, 55.45296096801758, 25.59769630432129, 0.6360974311828613, 28.90717124938965, 13.892430305480957, -27.729812622070312, -7.012378215789795, 6.852348804473877, 14.380492210388184, -47.19728088378906, 23.77581214904785, -56.592926025390625, 31.743389129638672, -29.857465744018555, 36.225364685058594, 31.396121978759766, -59.27637481689453, -26.58967399597168, -29.344430923461914, -48.96577072143555, 41.188961029052734, 7.659817695617676, 69.48136138916016, 9.641942977905273, -23.810827255249023, 46.376983642578125, 33.9498405456543, -26.843481063842773, 11.600346565246582, -23.449321746826172, 30.103561401367188, 41.40687561035156, -4.206896781921387, -30.733158111572266, 10.214044570922852, 5.417729377746582, 5.666568279266357, -17.232147216796875, -1.627461314201355, 18.330944061279297, -27.86248207092285, 12.275991439819336, 35.7778205871582, 13.863025665283203, -36.314090728759766, -0.9750432968139648, 7.735785007476807, 17.463411331176758, 44.57304763793945, 42.262813568115234, -0.6369906663894653, -36.60464096069336, -16.531068801879883, -4.812594413757324, -1.835119366645813, 3.262866735458374, -17.56702995300293, -29.22281837463379, 43.89979553222656, 37.466346740722656, -2.5703485012054443, 27.80327796936035, -38.49859619140625, -49.77651596069336, -4.972598552703857, -55.26194763183594, -12.656953811645508, 49.54469680786133, -20.86164665222168, 23.25003433227539, 68.48088073730469, 10.213423728942871, 0.6765551567077637, -27.993751525878906, 17.493545532226562, -21.093276977539062, 43.141624450683594, 35.84355926513672, -30.59955596923828, 18.3221492767334, -33.16929626464844, -7.437654495239258, -48.43803787231445, -3.044865608215332, -8.52209758758545, -4.770373821258545, -37.838897705078125, -28.652790069580078, 48.947998046875, 54.11268615722656, -53.15929412841797, -15.60075569152832, 16.26449966430664, -15.215169906616211, 34.625274658203125, -7.231308937072754, 27.26207733154297, -14.065361022949219, 35.45226287841797, 34.222713470458984, 24.39348602294922, 25.203222274780273, 47.21058654785156, -10.549833297729492, 31.21118927001953, -5.751341342926025, 23.669029235839844, 8.648307800292969, 15.428665161132812, 21.11928939819336, 33.69394302368164, 35.87840270996094, -39.87820053100586, 30.588594436645508, 8.78168773651123, 1.9181288480758667, 28.590641021728516, 16.333595275878906, 24.107603073120117, -4.460926055908203, 71.982666015625, -5.032452583312988, -22.532705307006836, -10.537199974060059, -18.36592674255371, -43.08340835571289, 6.944086074829102, 30.784217834472656, -13.965607643127441, 20.334049224853516, -0.06248569115996361, 58.22925567626953, -47.59956359863281, -63.292259216308594, -1.1892045736312866, -10.848254203796387, -45.70954895019531, -12.240245819091797, -18.412843704223633, 4.372364044189453, 21.470787048339844, -34.805416107177734, 7.630383014678955, -52.84635543823242, -66.25016784667969, -33.21104049682617, -38.26020431518555, 19.42957305908203, 33.32286834716797, 12.929908752441406, 55.99122619628906, -5.171237945556641, -4.05766487121582, -40.280033111572266, -15.545897483825684, -5.530770301818848, 20.952787399291992, 9.994955062866211, -45.28102493286133, -39.70393371582031, -6.379194259643555, 49.281307220458984, 42.736846923828125, -2.0235095024108887, -39.95631790161133, 5.9957661628723145, -55.5027961730957, 62.22422409057617, 4.240968227386475, -24.85148811340332, -13.536598205566406, -8.333507537841797, -12.403617858886719, -32.39713668823242, -76.57930755615234, 19.701597213745117, -50.28616714477539, 20.463293075561523, -103.03239440917969, -23.800945281982422, 23.10763168334961, 36.07911682128906, 26.28470230102539, -47.935604095458984, 8.000078201293945, -17.643695831298828, 6.654644012451172, -19.316360473632812, 27.063915252685547, -9.450932502746582, 38.22465896606445, -11.976021766662598, -39.14430618286133, 45.29057312011719, -32.729400634765625, -5.306967258453369, 9.978837013244629, 50.20096206665039, 15.480560302734375, 43.414344787597656, 10.260842323303223, 72.84969329833984, -4.27574348449707, -5.414602756500244, -11.403056144714355, 1.8566467761993408, -18.870800018310547, 15.179518699645996, -31.29840087890625, -16.13437271118164, 7.597033500671387, 7.276270866394043, -51.81051254272461, -16.32334327697754, -45.83074951171875, 44.73399353027344, 13.060340881347656, 7.56730842590332, 5.702342987060547, -44.77257537841797, 2.0458109378814697, 43.777259826660156, 5.1133341789245605, -1.5322693586349487, 11.90139389038086, 3.0300939083099365, 36.56859588623047, -16.28253936767578, 24.625988006591797, 46.976200103759766, -20.342037200927734, -18.615806579589844, -20.289405822753906, 15.976778030395508, 8.15785026550293, 21.415115356445312, -9.657248497009277, 21.13261604309082, -2.7106494903564453, -26.446918487548828, -38.497459411621094, 53.04685592651367, -16.194292068481445, -3.0890114307403564, -33.2432861328125, -8.09040355682373, -1.289534568786621, 25.058149337768555, -24.776927947998047, 8.602222442626953, 11.899701118469238, 21.1375789642334, 27.15364646911621, 73.01853942871094, -21.637958526611328, 13.889212608337402, -42.79835891723633, 50.60034942626953, -28.333995819091797, 27.92407989501953, 9.128539085388184, -9.649279594421387, 45.82565689086914, 0.18020543456077576, 42.31631088256836, 23.341461181640625, -33.03584671020508, 2.005239963531494, -19.324403762817383, 49.0324592590332, 19.1541805267334, 15.693290710449219, -27.399837493896484, 31.696149826049805, -49.597877502441406, -0.604073703289032, 1.099169373512268, -24.558673858642578, 13.937738418579102, 4.101775169372559, -1.1359407901763916, 4.830226898193359, 43.129459381103516, -16.052021026611328, 59.23385238647461, 48.00192642211914, -76.7889175415039, -10.29165267944336, 2.218114137649536, 31.100309371948242, -14.754188537597656, 8.830077171325684, -9.22115421295166, 42.20241165161133, 1.5676147937774658, -22.37177276611328, 68.73048400878906, -4.243480205535889, -31.17469596862793, 35.3853759765625, -22.925127029418945, -8.050507545471191, 17.19416618347168, -3.142024040222168, -11.131975173950195, -11.913749694824219, 4.386606216430664, -7.792789936065674, -22.92409324645996, -19.544771194458008, -43.93412780761719, 22.602262496948242, -43.719886779785156, 5.204146385192871, -29.750221252441406, -13.534259796142578, -8.38813591003418, -20.896163940429688, -37.61093521118164, 13.326007843017578, 3.1884756088256836, -38.35554885864258, 47.851444244384766, 39.3134880065918, -28.359987258911133, 39.55628204345703, 26.030532836914062, 8.285305976867676, -11.696928024291992, 0.5451962351799011, -4.584648132324219, 22.85700225830078, 22.82607078552246, -27.437114715576172, 2.735851764678955, 8.964818954467773, 31.02713966369629, -14.876652717590332, -51.258338928222656, 40.09959030151367, 37.37940979003906, 22.442800521850586, 13.220908164978027, -17.886980056762695, -5.640101909637451, -1.7461082935333252, 54.99962615966797, -6.684625625610352, 7.508543014526367, -16.57881736755371, -0.24434323608875275, 1.2596670389175415, 30.64916229248047, 9.575860977172852, 58.10318374633789, 36.03475570678711, 15.69083023071289, 92.23474884033203, -27.554096221923828, 30.71548080444336, -2.7022809982299805, 59.71717071533203, -39.7064094543457, 4.3045854568481445, -1.1073222160339355, -9.168810844421387, -28.919584274291992, 18.555749893188477, 94.05193328857422, -17.093563079833984, 55.84694290161133, 31.167329788208008, 30.1944637298584, -32.355438232421875, -39.21632385253906, -10.41808795928955, -19.618316650390625, -26.259765625, -14.14893913269043, -43.69709014892578, 37.19347381591797, 15.549827575683594, -24.91959571838379, 18.97475814819336, -33.80487823486328, 23.466711044311523, 6.6751532554626465, 2.8647537231445312, 32.055946350097656, 5.549980640411377, -56.41228485107422, -48.95808029174805, -28.382553100585938, -39.787353515625, -63.10496139526367, 30.384746551513672, -1.8606016635894775, -11.004681587219238, -48.43881607055664, 39.47876739501953, 35.351905822753906, 34.491607666015625, -11.062602043151855, -16.1672306060791, 31.975250244140625, -24.52122688293457, -10.526786804199219, -5.00913143157959, -5.454782485961914, -4.870365619659424, -5.524240970611572, 2.917013645172119, 28.27713394165039, -35.943538665771484, 44.76222229003906, 12.939298629760742, 5.994663238525391, -0.09004335105419159, 24.051223754882812, 12.674424171447754, -54.58378219604492, -20.404991149902344, -13.827322959899902, -3.3569226264953613, -16.068450927734375, 30.74541473388672, -36.381256103515625, 6.32186222076416, -24.457372665405273, -54.063087463378906, 89.82231903076172, -35.442955017089844, 23.3814754486084, 32.9559440612793, 50.0049934387207, -24.855207443237305, 20.420438766479492, 12.184471130371094, 13.935282707214355, 35.577205657958984, 20.13068199157715, 38.8436279296875, 64.85906219482422, 3.1333060264587402, 1.8520612716674805, 10.0131196975708, -5.245325088500977, -28.934486389160156, 68.17887878417969, 49.27095031738281, 49.445987701416016, -73.6584243774414, -2.8654394149780273, 40.52249526977539, -6.65139102935791, -17.915050506591797, 33.80714797973633, -11.713321685791016, 3.0421512126922607, 9.287251472473145, 19.2604923248291, -18.44029426574707, -19.678266525268555, -6.556521415710449, -51.037620544433594, 7.435401916503906, -40.18357849121094, 53.40231704711914, -22.8268985748291, -30.651094436645508, -9.21607780456543, -46.139827728271484, -69.01080322265625, -42.650604248046875, -36.34468078613281, -29.023284912109375, 16.75177574157715, 6.176065444946289, -30.406551361083984, 36.595054626464844, 11.20591926574707, 49.017276763916016, 23.219730377197266, 67.27308654785156, -1.9255558252334595, -11.17910385131836, -12.13550090789795, 9.568137168884277, 16.883005142211914, 15.563521385192871, 40.49087905883789, 1.2746962308883667, 18.234554290771484, -26.90904426574707, -44.6110954284668, 23.496809005737305, -46.831825256347656, 19.613325119018555, 15.612614631652832, -48.291988372802734, 72.54080200195312, -11.971622467041016, 14.939743041992188, -42.53615188598633, 30.037622451782227, -34.10203552246094, -12.668659210205078, -5.597914695739746, -13.958353996276855, 31.427095413208008, -7.0512919425964355, 25.37633514404297, 42.65817642211914, 23.887203216552734, -11.958125114440918, -12.812788963317871, -16.215898513793945, -9.546464920043945, -32.21549606323242, 7.02392053604126, -18.928770065307617, 32.28200912475586, 27.315948486328125, 5.212079048156738, -46.12017822265625, 21.15199851989746, -34.9089469909668, -42.920310974121094, 14.783141136169434, 34.63319396972656, 8.88458251953125, -0.17951612174510956, -43.62569808959961, -10.834291458129883, 29.888240814208984, 47.301483154296875, -28.875181198120117, 43.56357192993164, 31.142274856567383, -5.635378837585449, -40.247039794921875, 4.678833484649658, -26.898130416870117, 19.728355407714844, -16.564069747924805, -65.891357421875, 7.584665775299072, 3.950357437133789, 15.54831314086914, -9.84587574005127, 38.55619430541992, 31.803707122802734, 25.00988006591797, 3.6204001903533936, 16.82272720336914, -47.81435012817383, 24.87389373779297, -7.081677436828613, -19.1656436920166, -27.352638244628906, -22.113496780395508, 59.57132339477539, 2.0005669593811035, 17.115568161010742, 72.0485610961914, -3.884972333908081, -83.4537582397461, 49.213836669921875, 69.28467559814453, -9.034940719604492, 34.05243682861328, 2.1663589477539062, -7.868008136749268, 34.062957763671875, -27.589832305908203, -49.60991668701172, 17.714330673217773, -1.5139884948730469, 6.6474528312683105, 48.125709533691406, -9.99077033996582, -10.59268856048584, -89.62122344970703, -3.500187397003174, 30.708463668823242, 32.061279296875, -32.73663330078125, 11.941084861755371, -1.6687674522399902, -50.40911102294922, 23.948942184448242, -50.86202621459961, -50.602943420410156, 10.870925903320312, -5.462344646453857, 10.634995460510254, 16.090007781982422, 12.059877395629883, 15.75905990600586, 21.530534744262695, 27.317306518554688, -3.4079391956329346, -51.351051330566406, -21.080461502075195, 3.4977867603302, 16.192575454711914, 9.519831657409668, -9.820240020751953, -2.4893035888671875, -11.944459915161133, 15.535599708557129, 3.569361448287964, 31.664743423461914, -42.280948638916016, 48.68205261230469, 30.75314712524414, -13.195367813110352, 23.48960304260254, -17.855512619018555, -38.88766098022461, -27.334482192993164, 29.767837524414062, -33.37146759033203, -47.40666198730469, 12.576078414916992, 13.300070762634277, -62.08195114135742, -2.1716339588165283, 8.098607063293457, 19.123497009277344, 26.511472702026367, 47.88574981689453, -1.1126351356506348, -17.657453536987305, 9.62101936340332, 39.858829498291016, -38.46820068359375, 7.09959602355957, -17.522050857543945, 25.6903076171875, 25.875228881835938, 10.735684394836426, -25.03093719482422, -67.88027954101562, 2.5614864826202393, -10.229812622070312, 34.16084289550781, 12.158926963806152, 13.025947570800781, -4.173710823059082, 5.744342803955078, 41.17336654663086, 13.920151710510254, 8.70549488067627, 54.77756118774414, -1.1279621124267578, 30.780004501342773, 13.606904029846191, 4.485958099365234, 1.5671439170837402, -17.965038299560547, -13.914097785949707, 5.827916622161865, -4.605654239654541, -64.13522338867188, 38.01559066772461, 33.31909942626953, -71.7401351928711, -11.106830596923828, -10.762301445007324, -33.46332550048828, 38.302276611328125, -21.872114181518555, 32.322235107421875, -26.412565231323242, -24.52628517150879, -26.042503356933594, -10.035983085632324, -78.04985046386719, 30.135465621948242, 1.956005334854126, -15.69705581665039, -7.732632637023926, -14.70501708984375, 2.794698476791382, 55.90217971801758, 29.902198791503906, -38.756099700927734, 44.215885162353516, -49.74366760253906, -29.514633178710938, -34.83091735839844, 9.955924987792969, 2.050546646118164, 26.47696876525879, -11.406225204467773, 30.046022415161133, -7.256239414215088, -3.7463507652282715, -72.62481689453125, -4.741501808166504, -14.602783203125, 10.947013854980469, 40.936744689941406, -39.39997863769531, 49.46759796142578, 17.559179306030273, -52.59112548828125, 34.184120178222656, 28.7012882232666, 14.340285301208496, -35.639190673828125, -82.38710021972656, 51.99592590332031, 5.495556831359863, -7.11577844619751, -45.224082946777344, -34.641639709472656, 6.704947471618652, 18.686054229736328, -44.213836669921875, 1.8932576179504395, 11.409952163696289, 4.080273151397705, 11.379395484924316, -21.107837677001953, -38.73292922973633, 33.93471908569336, 16.397785186767578, -38.01677322387695, 2.9460909366607666, -15.427733421325684, -5.0872344970703125, -1.293586254119873, -22.04863929748535, -11.384011268615723, -24.169086456298828, -7.010481357574463, -16.653730392456055, 11.704401969909668, -19.024723052978516, -10.24326229095459, -28.301597595214844, -17.084327697753906, -90.0603256225586, -28.637510299682617, 5.746750831604004, 27.896379470825195, -20.69352912902832, 16.63028907775879, -37.37956619262695, 6.474887847900391, -33.527950286865234, 11.637075424194336, 19.806459426879883, 55.46521759033203, 35.24016571044922, -38.45351028442383, -0.4109835922718048, 14.077903747558594, -39.134185791015625, -41.819313049316406, 26.07966423034668, 1.13916015625, 25.91473388671875, 4.289540767669678, 36.237571716308594, -47.686649322509766, -15.820940017700195, 24.909849166870117, -7.791989803314209, 23.927207946777344, -21.435302734375, -14.951513290405273, -9.096685409545898, -9.390042304992676, -9.747827529907227, -35.20321273803711, -18.34458351135254, 12.980562210083008, 38.7689323425293, 41.3498649597168, -47.05218505859375, -36.686370849609375, 39.29933166503906, 31.63021469116211, 13.409757614135742, 28.313941955566406, -8.047943115234375, -4.25809383392334, -2.790093421936035, 15.788232803344727, 2.08990216255188, 26.104421615600586, 63.19666290283203, -68.36007690429688, 8.467185020446777, -7.222994327545166, 6.0106000900268555, 30.286396026611328, -21.905921936035156, -18.2110538482666, 42.20240783691406, -54.01817321777344, 6.502659797668457, 23.910812377929688, 24.297529220581055, 34.37925720214844, -36.34021759033203, -10.015689849853516, 77.61588287353516, -42.073387145996094, -21.066940307617188, 3.7098562717437744, -39.5454216003418, -27.316553115844727, -18.368268966674805, 32.05192184448242, 19.73813819885254, 49.05843734741211, -43.68450164794922, -7.7346367835998535, 15.53066635131836, 14.950521469116211, 56.45284652709961, 10.006368637084961, 2.5876035690307617, 18.88945770263672, -10.22374439239502, 59.988006591796875, -14.082062721252441, -37.32292938232422, 2.3422017097473145, 8.283607482910156, -38.42856216430664, 3.8658835887908936, -29.332300186157227, -47.150245666503906, 35.82358169555664, 2.4409117698669434, 5.060050010681152, 48.59952926635742, 9.796818733215332, 14.45079231262207, 19.82263946533203, 7.317981719970703, -44.259586334228516, 51.09114456176758, 17.592817306518555, 31.339277267456055, 5.472935676574707, -72.2061767578125, -47.4808235168457, 10.513059616088867, 88.71302032470703, 25.29604721069336, -1.9350953102111816, -18.197240829467773, 38.56418228149414, 10.780875205993652, 10.895822525024414, 26.36760711669922, 0.9323487281799316, 10.347139358520508, -22.887094497680664, -48.79887390136719, -43.50926208496094, -16.674907684326172, -2.2273056507110596, -40.006317138671875, 17.650653839111328, 53.14115524291992, 54.37946319580078, -21.480831146240234, 67.14776611328125, 16.6232852935791, -63.030494689941406, 59.37538146972656, 7.6135172843933105, 21.779273986816406, -5.083761215209961, 41.93404769897461, 27.055248260498047, -10.111896514892578, 10.568989753723145, 7.448696613311768, 40.285884857177734, -7.324557304382324, 80.24999237060547, 9.102753639221191, -5.478892803192139, 49.695274353027344, 26.22096824645996, -2.936760187149048, -27.432096481323242, 2.416527032852173, -11.175484657287598, 41.61555480957031, -20.704906463623047, -20.052431106567383, 9.603590965270996, -28.609621047973633, 29.15485382080078, -1.8297067880630493, -9.137710571289062, -9.5385160446167, 24.959192276000977, -22.844886779785156, -6.538424015045166, -19.497848510742188, -39.51425552368164, 2.0190963745117188, -38.00991439819336, -45.18495178222656, 55.33231735229492, 21.464406967163086, -49.29399490356445, -9.246085166931152, -20.977840423583984, -11.238943099975586, -36.309364318847656, 12.007081031799316, 15.906379699707031, -25.00360107421875, -47.27281951904297, -31.488393783569336, 12.314120292663574, -5.180263042449951, 21.28072738647461, -8.240338325500488, 40.776275634765625, 35.30595016479492, -16.804656982421875, 22.88941192626953, -38.46026611328125, -63.157859802246094, -29.50286293029785, -3.55863094329834, 11.806048393249512, 41.64966583251953, 2.069349765777588, 1.2519630193710327, -36.510765075683594, -27.1436824798584, -16.360292434692383, -31.241777420043945, -14.890284538269043, -42.799415588378906, 18.744258880615234, 1.4904453754425049, -24.67045021057129, -0.6745290756225586, -17.517295837402344, 9.710448265075684, -7.921353340148926, -49.92947006225586, -3.522566080093384, -9.712593078613281, 11.17343807220459, 17.138132095336914, -32.76812744140625, -31.385425567626953, -9.216313362121582, -21.39344596862793, 6.69415807723999, 27.220672607421875, 3.881214141845703, -4.868613243103027, -3.215289354324341, -4.291132926940918, -35.13035583496094, 27.539020538330078, 14.615354537963867, 50.877357482910156, -36.00815963745117, 0.7028113603591919, 19.27566146850586, 15.791877746582031, -6.473154544830322, -37.91027069091797, -16.801429748535156, 17.78023338317871, 4.999790668487549, 26.658832550048828, -3.471066474914551, 7.046723365783691, 17.590686798095703, 0.48798149824142456, 37.80670928955078, -41.05893325805664, -29.46057891845703, -2.8129587173461914, -38.96538543701172, 23.261838912963867, -17.168800354003906, 52.528804779052734, -8.419770240783691, 44.95827102661133, 24.492462158203125, -22.043506622314453, -47.89366149902344, 6.168598175048828, -20.461715698242188, -57.233802795410156, -24.229446411132812, 35.522422790527344, 9.442224502563477, 16.655778884887695, 30.663135528564453, -23.882322311401367, 26.459449768066406, -25.568195343017578, 19.635974884033203, 16.37202262878418, 5.918096542358398, 3.454746961593628, -26.565807342529297, 25.313241958618164, -52.648353576660156, 13.404566764831543, 46.18904113769531, 37.495338439941406, -9.459331512451172, -7.850182056427002, 22.785871505737305, 35.335472106933594, 15.48099136352539, -51.186073303222656, 42.80967712402344, -21.901874542236328, -76.69935607910156, -23.499656677246094, 6.637543201446533, 37.64316940307617, -26.99505615234375, -21.8464298248291, 24.391870498657227, 20.572214126586914, -8.021769523620605, 17.950153350830078, -21.376434326171875, -20.40821647644043, 13.141560554504395, 10.654557228088379, -10.71224308013916, 42.55343246459961, 26.601966857910156, -8.53692626953125, 31.652490615844727, -11.165047645568848, -9.0413236618042, -0.7734763622283936, 0.7819719314575195, -28.861597061157227, 23.51280975341797, 25.492475509643555, 11.442798614501953, -6.820449352264404, 30.390865325927734, 1.1019067764282227, -41.551055908203125, -18.444374084472656, 34.43054962158203, -59.651126861572266, 50.20294952392578, -4.0955119132995605, -37.59543228149414, 7.094361305236816, -44.63893127441406, 1.7138025760650635, 56.519630432128906, 3.1127564907073975, -17.852609634399414, -44.5833854675293, 24.408464431762695, -24.46221160888672, 55.41270446777344, 8.680839538574219, -2.3578736782073975, -23.66295623779297, 63.34047317504883, -51.506187438964844, 53.255531311035156, 39.26765823364258, -19.372177124023438, 17.13539695739746, 48.16404342651367, 10.090986251831055, 13.66177749633789, 14.052107810974121, 21.79767608642578, -17.63698387145996, 7.941510200500488, 37.07554626464844, -5.253480911254883, 12.9970121383667, 46.09941101074219, 23.910236358642578, -9.035224914550781, -23.540746688842773, -9.762089729309082, -7.713247299194336, -62.63930130004883, 63.809810638427734, 19.127138137817383, 2.3221733570098877, -19.080352783203125, -0.6688855886459351, 45.10093307495117, -0.6854103803634644, -13.98928165435791, 19.687841415405273, 23.14249610900879, 6.080225944519043, -41.611778259277344, -18.90802574157715, 28.092947006225586, -40.17107391357422, 48.02185821533203, 29.38328742980957, -6.458015441894531, -56.394466400146484, 14.414959907531738, 0.6004121899604797, 25.521467208862305, 19.207233428955078, 8.964850425720215, 26.500293731689453, -28.479394912719727, 5.229496002197266, 28.996803283691406, 12.05339527130127, -21.231155395507812, -12.123037338256836, -8.698141098022461, 31.900344848632812, -41.982845306396484, -21.570892333984375, 33.34976577758789, -5.036439895629883, 8.851076126098633, -19.855514526367188, 3.451139450073242, 20.429798126220703, 20.331722259521484, -5.106864929199219, 37.514278411865234, 15.439896583557129, 34.75004196166992, 32.440391540527344, -22.307161331176758, -0.7257152795791626, -20.843585968017578, -4.211066722869873, 29.62363624572754, 11.82809066772461, 39.20067596435547, 32.29088592529297, -60.36064529418945, -2.3918299674987793, -18.722150802612305, -7.041009902954102, -8.294631004333496, -38.641780853271484, -29.729373931884766, 16.570289611816406, -32.411075592041016, -13.782718658447266, 7.231268882751465, -41.10325622558594, 10.431154251098633, 59.296417236328125, 3.9949951171875, -18.80158042907715, 42.803504943847656, 12.771821022033691, -5.194753646850586, -51.93120193481445, 23.282447814941406, 13.28798770904541, -19.864967346191406, 46.50838088989258, -31.301883697509766, 22.755420684814453, 17.85980224609375, 11.91933536529541, 37.0445671081543, 18.23543930053711, -19.168367385864258, 16.1610164642334, -16.812192916870117, 4.671595573425293, 19.44005584716797, 0.5507205128669739, 31.88288116455078, 94.56550598144531, -44.9876594543457, 10.92422866821289, 3.533937454223633, 19.058069229125977, 37.53407669067383, 39.903621673583984, 28.531497955322266, -34.723785400390625, -17.333484649658203, -8.465313911437988, 34.094444274902344, -9.969483375549316, -42.418704986572266, -10.018078804016113, 44.31913757324219, 24.741439819335938, -11.437455177307129, 22.601036071777344, -18.36363410949707, -58.24842071533203, 21.609651565551758, 33.76203536987305, 14.32022476196289, -20.843538284301758, 37.16621398925781, -11.442253112792969, 24.551372528076172, 25.97816276550293, -17.967308044433594, -27.5050048828125, 26.036972045898438, -9.357264518737793, -20.60739517211914, -47.777183532714844, -9.565430641174316, 19.812211990356445, 16.595748901367188, 30.93015480041504, -27.05170249938965, 16.601797103881836, 42.33152770996094, -14.609718322753906, 16.883739471435547, -10.087776184082031, 42.07020568847656, -9.643476486206055, -2.8135242462158203, -14.530081748962402, 4.338172912597656, -11.080020904541016, -12.64786434173584, -1.7713159322738647, 23.1373348236084, -12.290623664855957, -0.1800120323896408, 20.402366638183594, -6.834218502044678, 4.99480676651001, 15.501173973083496, 34.87508773803711, 24.771677017211914, -3.1940078735351562, -36.051353454589844, -36.78251266479492, 10.465095520019531, 17.275863647460938, 1.1971098184585571, -10.589529991149902, -1.1255308389663696, -46.43422317504883, 42.70062255859375, -16.78956413269043, 31.66072654724121, 3.6923699378967285, -7.361914157867432, -55.887142181396484, -27.28877830505371, 36.2974853515625, 10.072906494140625, -29.50759506225586, 35.17827606201172, 0.9758279919624329, -17.33159637451172, 31.26432228088379, 59.14390563964844, -39.4785270690918, -18.390045166015625, 15.104713439941406, 43.81325912475586, 23.125547409057617, 26.454551696777344, -57.42812728881836, -33.931697845458984, 9.512789726257324, -6.932344436645508, -57.56974792480469, 4.67745304107666, -23.891815185546875, 17.40256118774414, 10.783455848693848, 29.98889923095703, 19.58702278137207, -31.806690216064453, -14.364195823669434, 32.9936408996582, 45.31712341308594, 28.077816009521484, -1.5849792957305908, 27.490774154663086, 26.95331382751465, -4.039674282073975, -13.063395500183105, -1.6481090784072876, 24.864913940429688, -16.93841552734375], [29.750518798828125, 16.527130126953125, -21.586631774902344, -28.014297485351562, 36.742645263671875, -37.240264892578125, 19.680198669433594, -27.622695922851562, 65.49512481689453, -33.96430969238281, -56.304141998291016, 42.14137268066406, -16.953838348388672, 48.36161804199219, -26.52869987487793, -9.530072212219238, -20.16183853149414, -28.37753677368164, 34.71894454956055, 62.12660598754883, 45.939388275146484, 46.16378402709961, -11.869379043579102, -25.55841064453125, 33.368194580078125, -18.795616149902344, 8.262880325317383, 37.80327606201172, -8.765677452087402, 15.166298866271973, -36.01588821411133, 80.40898895263672, 17.271902084350586, 24.818819046020508, -24.273000717163086, 39.35298538208008, -31.799251556396484, 41.691444396972656, 16.462295532226562, -18.167282104492188, 37.958595275878906, 1.1509629487991333, -4.6900811195373535, 35.650733947753906, -7.830751419067383, -10.17359733581543, 13.603822708129883, 29.35129165649414, -58.11910629272461, 27.983566284179688, -35.453758239746094, -34.641788482666016, -3.8105549812316895, 14.14453411102295, 40.2004508972168, -10.162497520446777, -42.655235290527344, 44.838871002197266, 8.923783302307129, -19.47259521484375, -10.74130630493164, -71.31117248535156, -9.489974021911621, 71.00224304199219, -37.70178985595703, -27.948062896728516, -48.30293655395508, -5.623702049255371, 2.810441017150879, 60.06134796142578, 22.371089935302734, 12.367846488952637, -14.609949111938477, 3.8112540245056152, 3.127060651779175, -45.53115463256836, -40.592681884765625, 27.684770584106445, -13.585480690002441, 10.821120262145996, -5.610238075256348, -16.32804298400879, 12.512983322143555, 7.201376438140869, 1.80006742477417, -72.36615753173828, 38.49406814575195, -9.430412292480469, 27.186676025390625, 12.812432289123535, -42.0848274230957, 22.21589469909668, 96.94216918945312, -9.537761688232422, 19.65299415588379, -49.032127380371094, -17.632474899291992, 39.48123550415039, -3.035447120666504, -0.8057662844657898, 47.32089614868164, -23.297590255737305, -19.98924446105957, 6.413485050201416, 16.585651397705078, 21.697437286376953, 2.695835590362549, 45.03080368041992, -79.08849334716797, -50.61002731323242, 21.507610321044922, -22.22216033935547, 35.46699523925781, -27.614696502685547, 26.136173248291016, -6.562570571899414, -11.258774757385254, -2.4944238662719727, -7.020579814910889, -10.342084884643555, -3.5049338340759277, -21.405014038085938, -17.878461837768555, 24.025850296020508, 12.201533317565918, -38.50578308105469, -27.016014099121094, 30.853164672851562, -4.0572733879089355, -26.86467742919922, 21.635934829711914, -67.72085571289062, 29.139881134033203, 7.437102794647217, -6.486784934997559, -36.49968719482422, 15.932051658630371, -29.892955780029297, 7.101541519165039, 16.5191707611084, 20.46575355529785, -7.840105056762695, -32.29729080200195, -1.8986384868621826, 48.400699615478516, -8.837698936462402, 11.314199447631836, 8.978466033935547, -113.88385009765625, 17.621000289916992, -16.054826736450195, -33.74213409423828, 8.890738487243652, -1.9858990907669067, -64.79753875732422, 63.03182601928711, 33.75049591064453, -17.598045349121094, 14.337779998779297, 55.89789962768555, 1.2882287502288818, 3.1172780990600586, 16.350399017333984, 14.784675598144531, -19.652341842651367, 40.1075439453125, -16.769990921020508, -5.133139133453369, -39.62487030029297, 3.539137601852417, -1.2752271890640259, 36.812599182128906, -23.416460037231445, 0.183742955327034, 17.653526306152344, 27.03681182861328, 16.897865295410156, 2.6541128158569336, -64.51741790771484, -59.71822738647461, 32.402950286865234, -21.021272659301758, -20.35704231262207, 9.39892578125, -31.851253509521484, -42.5068244934082, 15.518519401550293, -25.93997573852539, -71.08611297607422, 33.53383255004883, 21.509681701660156, -15.196924209594727, -1.6521962881088257, -16.99312973022461, 15.91866397857666, -25.91973114013672, -32.516326904296875, -21.67803192138672, -50.28317642211914, 54.597450256347656, -39.58789825439453, 4.498644828796387, 27.72601318359375, -18.775453567504883, -5.913801193237305, 8.730658531188965, 25.664058685302734, -14.378120422363281, -37.40934371948242, -19.75428009033203, -14.808073043823242, -9.392034530639648, 14.080801010131836, -17.154449462890625, 26.525739669799805, 19.91376495361328, 12.688604354858398, 16.327987670898438, -27.40436553955078, -25.879112243652344, 15.900768280029297, -9.152623176574707, -3.7315149307250977, 19.98177719116211, -22.30350685119629, -52.03602600097656, 28.52972412109375, -9.148334503173828, -26.980234146118164, 25.02805519104004, 10.041805267333984, -30.291955947875977, 21.619564056396484, 40.970462799072266, -10.09482479095459, 69.04889678955078, -2.1893861293792725, -15.573250770568848, 23.054519653320312, -7.1292219161987305, -13.850976943969727, 14.271208763122559, -4.948681354522705, 32.23837661743164, -12.15293025970459, -21.86700439453125, 35.92210006713867, 16.94008445739746, 34.56809997558594, 17.755155563354492, 10.3888521194458, 10.596186637878418, 24.683313369750977, 27.55818748474121, 33.674339294433594, -5.94276237487793, 7.1104841232299805, -5.819792747497559, -13.592280387878418, -16.205780029296875, 65.27561950683594, 17.776403427124023, -12.0831298828125, -36.42391586303711, 11.372491836547852, 7.1154866218566895, -34.598873138427734, 1.3957642316818237, 11.747736930847168, -28.07941436767578, -102.58580780029297, -23.349828720092773, 28.760292053222656, -9.099564552307129, -15.104576110839844, -12.264158248901367, -31.830385208129883, -39.26179122924805, 20.404294967651367, -8.996344566345215, 7.519759178161621, -30.16891860961914, -29.36627960205078, 8.179856300354004, 30.82855987548828, -44.097145080566406, -26.264707565307617, -6.475411891937256, -31.456554412841797, -9.225296020507812, 20.54347038269043, -11.464200019836426, 8.028449058532715, 40.79033660888672, 21.475923538208008, 0.5648046731948853, 40.71965789794922, -38.02883529663086, 17.0080509185791, -16.939071655273438, -55.53152084350586, 4.339676856994629, 35.67100143432617, -16.929162979125977, -19.5567626953125, -12.632145881652832, 10.149389266967773, 42.67887878417969, 63.37438201904297, 10.248714447021484, 11.140046119689941, 22.17674446105957, -5.780368804931641, 38.91043472290039, -0.022256996482610703, -14.378786087036133, 22.888181686401367, -30.043216705322266, -57.02421188354492, 15.153712272644043, -4.265240669250488, 51.21529769897461, -6.8417887687683105, -26.324424743652344, -25.630535125732422, 38.055999755859375, 17.09733772277832, 24.077070236206055, 10.40320110321045, -13.875929832458496, 4.181436538696289, 32.019134521484375, 0.29207831621170044, -41.06778335571289, -26.153465270996094, 33.2541618347168, -6.4586005210876465, 14.670209884643555, -19.73579978942871, 54.570777893066406, -4.789566993713379, 27.954124450683594, 6.269696235656738, -2.926645517349243, -8.090960502624512, -24.952547073364258, 10.801647186279297, -4.716063499450684, 13.05097484588623, -28.148723602294922, 27.74517822265625, -16.217144012451172, 19.90467071533203, 2.5730700492858887, -26.351430892944336, 32.87020492553711, -1.1153861284255981, 31.002599716186523, 43.03889465332031, 5.602787971496582, 1.9560719728469849, 35.384063720703125, 53.98166275024414, -9.72485637664795, 23.126161575317383, -31.794025421142578, -3.537205457687378, -1.567497730255127, 34.101715087890625, -56.738285064697266, -11.910199165344238, -18.15191650390625, -39.398834228515625, -22.317041397094727, 19.168624877929688, -6.861213207244873, -24.462839126586914, -7.1396164894104, -10.18052864074707, -10.32833480834961, -42.78304672241211, 84.77533721923828, -43.610477447509766, 31.160030364990234, 29.367311477661133, -26.472476959228516, -4.167179584503174, 13.6826171875, 13.752433776855469, 9.858620643615723, 20.73253631591797, 10.106605529785156, -2.6097159385681152, 10.082852363586426, 23.623472213745117, -0.47181180119514465, 0.00912911631166935, -18.70726776123047, -15.216164588928223, -16.590354919433594, -37.673683166503906, 10.22769832611084, -17.910419464111328, -30.117219924926758, 14.091292381286621, 11.061423301696777, -31.914764404296875, 18.487213134765625, 52.32889938354492, -1.3108755350112915, -17.17303466796875, 50.95952606201172, -5.601097106933594, -20.475563049316406, 11.59036636352539, -18.930923461914062, -52.77552032470703, 38.74674606323242, -21.547603607177734, 35.029205322265625, 16.335378646850586, -18.94882583618164, -3.942950963973999, 21.793851852416992, 37.62847137451172, -44.23338317871094, -54.75708770751953, -24.371265411376953, -66.06757354736328, 12.502552032470703, 13.892581939697266, -19.900156021118164, 22.62511444091797, -25.069181442260742, -50.05317306518555, 20.79513931274414, 6.290126800537109, 23.096281051635742, 0.8204867243766785, 10.49377155303955, -4.721114158630371, 49.79692459106445, 13.070810317993164, -34.91763687133789, -15.126038551330566, 28.64529800415039, 3.814128875732422, -50.098243713378906, -49.21852493286133, -22.721752166748047, 3.764075756072998, -21.93858528137207, -12.149624824523926, 12.18928050994873, -53.839481353759766, -38.218833923339844, -23.58194351196289, 34.08717727661133, -0.6944848299026489, -21.72039031982422, -4.121978759765625, 35.58763885498047, 4.9563984870910645, 14.853130340576172, -13.735492706298828, 7.469149589538574, -16.871017456054688, 13.829814910888672, 2.851886510848999, -6.227846622467041, 5.021266460418701, 39.41293716430664, 2.5473475456237793, 23.839935302734375, -23.41939353942871, -28.360387802124023, -33.77973175048828, -41.31229782104492, 1.1844180822372437, 49.76188659667969, 22.834095001220703, 2.3280692100524902, -19.84327507019043, -37.328102111816406, -38.736690521240234, -23.249059677124023, 23.853805541992188, -14.253944396972656, 18.63043212890625, -52.16429138183594, 16.215232849121094, 8.80267333984375, 65.51652526855469, 33.179344177246094, -49.012271881103516, 12.610905647277832, 6.973759651184082, -18.701955795288086, 55.749977111816406, -38.00923156738281, 30.153518676757812, -30.346036911010742, 54.68318176269531, 62.82118225097656, -4.858864784240723, 2.9065542221069336, -2.4303219318389893, -0.8669081330299377, 32.66044998168945, 21.73170280456543, -43.0118522644043, 1.4256932735443115, 8.48311710357666, -12.39073657989502, -6.312501907348633, -10.04533576965332, 38.732269287109375, -12.377029418945312, -28.95549774169922, 22.284828186035156, 17.42411994934082, -4.504857540130615, -5.603884696960449, 31.71133804321289, -13.89714241027832, 14.055610656738281, -38.96015548706055, -24.382709503173828, 1.2576701641082764, -41.064002990722656, 26.864070892333984, -23.48426628112793, -17.81618309020996, 34.756134033203125, 16.661579132080078, -22.25019645690918, -17.156810760498047, -29.90325927734375, -42.8430061340332, 11.8328275680542, -68.88575744628906, -3.927884578704834, 74.4233627319336, 0.28134116530418396, -3.1936302185058594, 1.0363850593566895, -36.73260498046875, 47.82294845581055, 5.2135772705078125, 9.738767623901367, 19.66883087158203, -26.2345027923584, 32.61161804199219, 7.2073211669921875, 35.3072624206543, -14.413223266601562, -55.749000549316406, 14.442414283752441, -16.392026901245117, 7.676809310913086, 41.96464538574219, 20.39899444580078, 24.204570770263672, -3.843647003173828, 34.507415771484375, -14.917778015136719, 1.7230926752090454, -19.07541275024414, 9.430387496948242, -17.94428825378418, 40.083370208740234, -87.31785583496094, -57.55577087402344, 44.500789642333984, 19.418930053710938, 24.450542449951172, -54.427101135253906, 48.6295051574707, -48.194793701171875, 13.936891555786133, -9.122926712036133, 14.77777099609375, -37.19818878173828, 42.495235443115234, -16.974401473999023, 40.97380065917969, 40.513427734375, -23.093717575073242, -15.616781234741211, 18.305095672607422, 12.31607723236084, 19.370664596557617, -4.435492515563965, -31.88881492614746, 12.381355285644531, 6.662533283233643, -38.99850845336914, 69.28086853027344, 6.359117031097412, 4.493412017822266, -34.11550521850586, 25.044979095458984, -12.795989990234375, 21.6762638092041, -23.500974655151367, -16.395343780517578, -43.75679397583008, -56.07969284057617, -22.494237899780273, 30.65273094177246, -19.626081466674805, 21.782241821289062, 49.105411529541016, -32.77928161621094, -17.368249893188477, -34.286495208740234, -12.478652000427246, 1.7152092456817627, 1.8286124467849731, -38.00339126586914, -24.70606231689453, -0.5381014347076416, 20.555877685546875, -6.44069242477417, 34.79054641723633, -6.939378261566162, 56.03211212158203, 24.126684188842773, -8.571744918823242, -44.322200775146484, -23.34180450439453, -28.988088607788086, -15.848493576049805, -11.616100311279297, -21.53049659729004, -13.943833351135254, 34.507625579833984, 0.33775272965431213, 39.13428497314453, 6.866544246673584, -13.8104887008667, -6.993286609649658, -1.494663119316101, -12.230170249938965, 18.905309677124023, 8.391093254089355, -30.873870849609375, 4.835331916809082, 27.665199279785156, -28.905061721801758, -4.47214937210083, -10.597978591918945, -28.7327880859375, -4.2494306564331055, 5.132984161376953, -16.526386260986328, -24.7269229888916, 2.2315969467163086, -20.261381149291992, 32.990169525146484, 44.32255935668945, -31.21478271484375, -11.068079948425293, -2.752915620803833, -23.731542587280273, 19.873071670532227, 28.166698455810547, 1.720703125, -7.400574207305908, -31.123821258544922, 19.829906463623047, 31.834552764892578, 1.094934105873108, 16.050159454345703, 51.675926208496094, 39.13175582885742, 45.75274658203125, -47.39695739746094, 28.247095108032227, 7.74351692199707, -13.944220542907715, -21.205860137939453, -32.56650924682617, 18.661151885986328, 13.26989459991455, -3.9739573001861572, -18.300703048706055, 11.25155258178711, -34.12419509887695, 3.528874397277832, 31.188396453857422, 27.579914093017578, -22.466386795043945, 18.427061080932617, 5.694864273071289, 68.528076171875, -42.30616760253906, -19.777095794677734, -7.44973087310791, -45.035186767578125, -3.136913299560547, 49.796875, -13.126721382141113, -21.128231048583984, -38.81452560424805, -3.475339651107788, -43.679256439208984, -24.73084259033203, -6.275285720825195, -16.052331924438477, 40.596622467041016, 74.11102294921875, -25.61502456665039, 50.2340202331543, 7.458212852478027, 9.742029190063477, -0.5538027882575989, 27.642545700073242, -57.29015350341797, -0.6264234185218811, -5.5324602127075195, 3.9112257957458496, -58.25951385498047, -10.281940460205078, -7.529599189758301, -52.835166931152344, -22.34950065612793, 13.64008903503418, 41.00320816040039, -0.33454132080078125, 37.83402633666992, -5.27941370010376, -19.50575065612793, -47.33747100830078, -33.353912353515625, 0.607895016670227, 14.129005432128906, -53.24570083618164, 39.707366943359375, 62.532772064208984, -11.724555969238281, 54.532257080078125, -13.564484596252441, -19.276445388793945, -15.805159568786621, -48.488037109375, -38.185760498046875, 27.672260284423828, 34.96626281738281, -32.51664352416992, -28.531614303588867, -2.143728256225586, -86.05863189697266, 18.288166046142578, 11.436490058898926, 3.2840049266815186, 0.6260094046592712, -24.01555824279785, -11.825190544128418, -6.038148880004883, 5.534732818603516, -21.766643524169922, 17.495101928710938, 31.42760467529297, -15.179616928100586, 24.685787200927734, 22.34034538269043, -29.77280616760254, 5.784495830535889, 33.36048126220703, -56.816864013671875, -60.494998931884766, -9.878673553466797, 1.1657620668411255, 41.29486083984375, -8.746914863586426, 19.307079315185547, -13.510358810424805, -5.741456508636475, -56.19965744018555, 6.946398735046387, 80.98163604736328, 11.025397300720215, -8.958907127380371, 5.378695964813232, -36.75154113769531, 23.440088272094727, -67.3404769897461, -16.194751739501953, -11.80605411529541, 4.58388614654541, -36.2805290222168, -7.982446193695068, -7.236290454864502, 9.173420906066895, 7.601919651031494, 10.277802467346191, 34.71007537841797, 4.278231143951416, -2.980529308319092, 65.90303802490234, 21.10198211669922, 49.906639099121094, 2.06160306930542, 30.46534538269043, -41.788917541503906, 18.204330444335938, 7.789425373077393, 1.6993374824523926, -19.80738067626953, 5.899982929229736, 6.688464164733887, 14.493950843811035, 10.422100067138672, -8.316845893859863, -13.516295433044434, -19.03244972229004, -38.759883880615234, -7.989718437194824, -32.4730339050293, -18.318681716918945, 4.86586332321167, -5.073174476623535, -8.284396171569824, 9.678171157836914, 22.01265525817871, 12.223569869995117, -4.702672958374023, -12.336337089538574, 19.00028419494629, -11.561756134033203, 8.994844436645508, 20.585590362548828, 37.49641799926758, 29.391393661499023, -26.39263343811035, -28.224750518798828, 13.435835838317871, -21.81205177307129, -3.615145206451416, -12.788329124450684, -14.300196647644043, -35.5283203125, 14.930456161499023, 17.7259464263916, -22.179523468017578, -43.89727020263672, 2.494659423828125, 30.51789093017578, 18.514768600463867, -22.617267608642578, 7.968592643737793, 0.6335572600364685, 2.8807246685028076, 20.32626724243164, 1.8714591264724731, -27.10198211669922, 51.17504119873047, 36.9822883605957, -43.231082916259766, 7.147892951965332, -5.930444717407227, 1.4756969213485718, -25.77520179748535, 33.26582336425781, -7.974783897399902, 35.66098403930664, 57.25001907348633, -12.527682304382324, 39.61040115356445, 32.29949951171875, -28.156763076782227, 6.227449893951416, 15.242494583129883, -12.192474365234375, -15.217000961303711, -0.45801523327827454, -11.891329765319824, -26.75385856628418, -19.841434478759766, -5.520999431610107, 19.05160903930664, 18.99333953857422, -13.098625183105469, 64.65415954589844, 12.719858169555664, 10.094674110412598, 4.18501615524292, 27.828359603881836, 34.84925842285156, -6.644758224487305, -18.524120330810547, 6.638417720794678, 18.516016006469727, 36.570396423339844, -34.80310821533203, 30.890480041503906, -5.337017059326172, -19.029788970947266, 17.116819381713867, 14.384078025817871, -28.258451461791992, 7.930007457733154, -3.2065787315368652, 9.491253852844238, 11.432547569274902, -38.215415954589844, 16.235273361206055, 39.611873626708984, 5.567775726318359, -32.300628662109375, 102.19319152832031, -36.75468826293945, 1.6241556406021118, 0.6442298889160156, -18.123531341552734, -29.356229782104492, 27.370819091796875, -41.42901611328125, 17.009838104248047, -1.7442107200622559, 13.211024284362793, 8.004080772399902, -16.838369369506836, 24.694520950317383, -36.54230880737305, 48.10417938232422, 4.288271903991699, -5.567305564880371, 17.02708625793457, 2.645862579345703, -26.421350479125977, 33.16667175292969, 10.928412437438965, 12.324953079223633, -7.66195011138916, 40.649085998535156, -10.168976783752441, -33.014556884765625, 6.610543727874756, 18.4871883392334, 42.07327651977539, -14.079930305480957, -26.1525936126709, 7.354012966156006, 47.86116027832031, -29.68545150756836, 41.02613830566406, -10.02957534790039, -9.121816635131836, -55.949920654296875, -5.504657745361328, -32.320091247558594, -3.2466623783111572, 82.72550964355469, -20.192235946655273, -26.724422454833984, 31.7627010345459, 6.171358108520508, -22.442535400390625, -71.14753723144531, -12.725224494934082, -54.63500213623047, -16.3604793548584, 18.156536102294922, -16.700580596923828, -88.81707763671875, -8.909383773803711, -9.654034614562988, -7.605669975280762, -12.102526664733887, -0.724268913269043, 47.49655532836914, 20.75692367553711, -1.1823304891586304, -29.372045516967773, -27.999303817749023, -51.564414978027344, -24.69363021850586, -4.140220642089844, 9.234962463378906, -2.6857264041900635, 29.844493865966797, -37.334903717041016, -25.205175399780273, -13.199392318725586, 0.9726969599723816, -4.0442376136779785, 36.294063568115234, 43.66073226928711, -20.371753692626953, -24.066844940185547, 24.251955032348633, -53.09446716308594, -10.424559593200684, -30.400489807128906, 5.3669939041137695, 56.95704650878906, 1.6847553253173828, 3.1885530948638916, 26.71318244934082, 31.20332908630371, -32.47293472290039, 7.721514701843262, 24.05929946899414, 3.4014294147491455, -1.9545185565948486, -6.733929634094238, 4.462260723114014, 64.05264282226562, 8.781246185302734, 33.76294708251953, 58.88090896606445, 43.684776306152344, 43.374515533447266, 3.2132484912872314, 13.454410552978516, -54.89547348022461, -33.69974136352539, 12.962416648864746, -42.82353210449219, -61.56053924560547, -39.974586486816406, 44.12571334838867, 21.994237899780273, 3.5336966514587402, 19.078113555908203, -0.05288819596171379, -35.56291198730469, -12.711797714233398, -18.194284439086914, -3.2138595581054688, -0.7709798812866211, -2.5012457370758057, 2.268789291381836, -28.790058135986328, -14.11711311340332, 46.36101531982422, 13.360926628112793, 10.577323913574219, -4.4864501953125, -10.546989440917969, 7.151930332183838, -15.789227485656738, -2.0041677951812744, 49.40005111694336, 26.23628807067871, 3.572373151779175, -29.82174301147461, -15.453901290893555, -13.610937118530273, 24.618383407592773, 19.938854217529297, 38.30736541748047, 28.921894073486328, -18.560365676879883, -0.812893271446228, 1.7177530527114868, -3.668534517288208, -17.83457374572754, 40.08433532714844, -3.571431875228882, 52.42751693725586, -13.615862846374512, 6.254678726196289, 21.20580291748047, 22.885536193847656, -12.564292907714844, -33.27559280395508, 16.025602340698242, 6.9562087059021, -53.2083854675293, -31.141098022460938, -10.643207550048828, 0.5058449506759644, 49.32880401611328, -48.18638229370117, -10.850231170654297, 2.7525134086608887, -82.87532043457031, -38.04667663574219, 44.625362396240234, -24.396221160888672, -3.3814733028411865, -15.628625869750977, -42.62685012817383, -11.465202331542969, 24.266265869140625, 27.587406158447266, -29.822404861450195, 58.96702194213867, -28.821638107299805, -29.400999069213867, 5.210236549377441, 38.18546676635742, 2.2322075366973877, -22.406524658203125, 44.255348205566406, 0.21554026007652283, -35.883140563964844, -14.800690650939941, 11.626437187194824, -6.7416253089904785, 52.751708984375, -14.211387634277344, -4.718666076660156, 7.702823162078857, 3.9750936031341553, 20.68813133239746, 13.556886672973633, -25.179351806640625, 47.21305465698242, -6.522112846374512, -19.29551887512207, -3.272388219833374, -62.507442474365234, 46.419193267822266, -29.92290687561035, -9.290046691894531, -38.8819580078125, -3.1406149864196777, 11.689824104309082, 41.2460823059082, 6.304861068725586, -23.801546096801758, -15.143049240112305, 32.82186508178711, 1.4489675760269165, -10.227505683898926, -39.95928955078125, 11.871125221252441, -6.703638553619385, 10.222341537475586, -2.642282009124756, -12.231661796569824, 13.00594711303711, -34.285797119140625, 32.74758529663086, 4.078609466552734, 1.9714951515197754, 59.12778854370117, -27.20902442932129, 6.559642791748047, -7.918606281280518, 7.930013656616211, 6.296032428741455, -5.566397666931152, -3.771702766418457, -20.705602645874023, 44.24767303466797, -10.343655586242676, 6.946366310119629, -28.115310668945312, 44.7774543762207, 46.9284782409668, 8.422436714172363, 7.520756721496582, -23.97062110900879, -45.264183044433594, -27.19430923461914, -22.012834548950195, 35.13223648071289, -44.2844123840332, 32.41928482055664, -13.286519050598145, 20.334287643432617, -19.69089698791504, -15.639265060424805, -13.124605178833008, 9.845703125, 6.3126630783081055, 16.05384063720703, -35.04454040527344, -21.749601364135742, 29.361764907836914, 28.306373596191406, -13.732040405273438, 2.3519744873046875, -42.4609260559082, 7.031232833862305, -15.427873611450195, 24.76654624938965, 53.47184371948242, -32.19221115112305, 21.53544807434082, 46.442901611328125, -74.49392700195312, -18.259986877441406, -9.448458671569824, 17.121173858642578, 15.152647972106934, 15.84748363494873, -52.694366455078125, 28.410781860351562, 7.267761707305908, 59.60941696166992, -15.740845680236816, -11.941614151000977, -35.73969650268555, 11.65284252166748, 7.996305465698242, 68.62598419189453, -48.500343322753906, 43.51133346557617, 16.876529693603516, -22.375038146972656, 26.53400230407715, 22.922216415405273, 7.000394344329834, -2.010890245437622, -0.33501747250556946, -12.166544914245605, -42.83019256591797, -26.468914031982422, -3.3357675075531006, -4.929112434387207, -32.50730895996094, 15.845248222351074, 34.934879302978516, 41.28311538696289, -3.143332004547119, -40.94648361206055, -7.58314847946167, 13.757216453552246, 34.424312591552734, 31.154987335205078, -48.93974685668945, -24.546297073364258, -18.225942611694336, 7.950882911682129, -23.627225875854492, -22.760822296142578, -8.170053482055664, -43.179542541503906, -10.774514198303223, 33.60162353515625, -38.533878326416016, -22.54888153076172, 21.109384536743164, -28.35310935974121, 38.26435852050781, -79.33210754394531, 31.83167266845703, -8.75227165222168, 59.57328796386719, 12.183572769165039, 38.83124542236328, 19.092578887939453, -15.85116195678711, -47.65631866455078, -25.551633834838867, 32.60226821899414, -36.65537643432617, -37.08842849731445, -52.940208435058594, 54.601741790771484, -1.4943821430206299, 28.806350708007812, 9.659135818481445, -10.362025260925293, -36.950496673583984, -5.298960208892822, 68.82970428466797, -0.4778871238231659, 19.353595733642578, 4.793064594268799, 11.498722076416016, -6.343329429626465, -18.70777702331543, 44.411258697509766, -10.864645004272461, 28.918668746948242, 61.46565246582031, -9.976644515991211, 11.291030883789062, 34.209754943847656, -12.808606147766113, 59.49635314941406, -22.369510650634766, -15.600777626037598, 21.8746337890625, 5.349743843078613, 30.111204147338867, -19.6180477142334, 13.2739839553833, 30.434593200683594, -19.424560546875, 29.713247299194336, -22.66157341003418, 50.438419342041016, -19.06845474243164, -11.685220718383789, 3.3647518157958984, 4.481256484985352, 7.3997602462768555, 8.219531059265137, 12.347251892089844, 2.083054542541504, 14.788094520568848, 16.577423095703125, 24.693742752075195, 5.191773414611816, 0.134458988904953, 12.936607360839844, -50.963802337646484, 21.663583755493164, -33.74047088623047, -9.794678688049316, 4.319575309753418, -27.199134826660156, 7.922262668609619, -4.946422576904297, 16.57880973815918, -32.28506851196289, -18.817943572998047, 8.516365051269531, -39.6072998046875, 4.2434844970703125, 32.74038314819336, 67.81099700927734, -2.532344341278076, -11.021598815917969, 39.413917541503906, 43.11530303955078, -42.688541412353516, 4.401670932769775, 14.838197708129883, -44.54768371582031, 27.644527435302734, 19.21463394165039, 5.303627014160156, 6.211668014526367, 45.618995666503906, -35.039241790771484, -39.401023864746094, 18.477365493774414, 32.77574920654297, -4.76947021484375, -6.576275825500488, -80.25277709960938, 4.646780967712402, 51.82379150390625, -14.264662742614746, 32.83155822753906, 76.41639709472656, -12.801067352294922, -24.050527572631836, -21.010194778442383, 53.76831817626953, 5.9553327560424805, 28.075969696044922, -6.849782943725586, 28.908306121826172, 38.23561477661133, -18.831689834594727, 13.662635803222656, -9.816483497619629, -13.025762557983398, -14.539420127868652, -21.593950271606445, 10.305493354797363, -7.160860538482666, 17.267288208007812, -25.22593879699707, 36.907081604003906, -18.955766677856445, -37.25046157836914, 4.514325141906738, -49.3704948425293, 35.71373748779297, -63.31865692138672, 79.83186340332031, 11.574972152709961, 27.907209396362305, -9.655503273010254, 16.903533935546875, 2.2494845390319824, -43.69599151611328, 29.49180030822754, -11.270406723022461, 12.846779823303223, -25.123687744140625, -12.26058292388916, 6.531310558319092, 34.81344985961914, 16.438335418701172, 10.926115036010742, 19.102121353149414, -27.832372665405273, 31.697656631469727, -15.320013999938965, -68.65950012207031, -70.6008529663086, -18.15719223022461, -1.3752456903457642, 33.6435546875, 0.2844783663749695, -12.983085632324219, -24.315731048583984, 47.21376419067383, -11.984214782714844, 7.7054009437561035, -35.02500534057617, -23.350727081298828, 25.900379180908203, -14.580491065979004, 54.87137985229492, 29.365663528442383, -15.387940406799316, 21.168609619140625, -6.4659929275512695, 27.83650016784668, 20.213653564453125, 26.50645637512207, 39.14429473876953, -6.172632217407227, -27.744953155517578, -9.384641647338867, 5.190624713897705, -27.588266372680664, 11.467529296875, -14.572328567504883, 30.423185348510742, 3.384108304977417, 4.2436041831970215, 7.579276084899902, -29.327707290649414, 16.709213256835938, -2.8438193798065186, 0.9964876174926758, -0.14451339840888977, 3.5358738899230957, -63.474117279052734, -23.727575302124023, -50.90018844604492, 0.6021029949188232, -27.109676361083984, -24.6257381439209, -2.730494260787964, -28.433683395385742, 39.3157958984375, -3.75307035446167, -9.88521671295166, 35.186946868896484, 47.424713134765625, 10.111491203308105, 21.994932174682617, -12.604302406311035, 13.368579864501953, -20.73189926147461, 13.997419357299805, 28.876134872436523, -21.816944122314453, 6.1028947830200195, 27.736591339111328, 9.471245765686035, -13.79580020904541, 39.787986755371094, -2.8530972003936768, 24.190336227416992, -32.90424346923828, -37.144004821777344, -16.305810928344727, 1.4103374481201172, 24.30962562561035, -29.289003372192383, -24.239866256713867, 39.287879943847656, 31.541446685791016, -19.1664981842041, -8.45726490020752, -12.096819877624512, 20.822328567504883, 10.858853340148926, -35.10321044921875, 40.70603942871094, 55.78883361816406, -17.132843017578125, -21.3778133392334, -31.231040954589844, -1.5680261850357056, -29.638492584228516, -35.16229248046875, -63.13667678833008, -11.091134071350098, 32.711936950683594, 26.43964958190918, -14.086423873901367, -34.871009826660156, 17.172924041748047, 15.832181930541992, -77.35125732421875, 17.66994857788086, 11.902045249938965, 17.959138870239258, -13.027276039123535, -7.589431285858154, 11.207804679870605, 14.075796127319336, 33.914794921875, -6.112236022949219, 8.684860229492188, -11.704678535461426, -17.358835220336914, 42.32914733886719, -32.96912384033203, -29.56650161743164, 63.34220504760742, 58.272579193115234, -64.61162567138672, 71.9173812866211, 0.5507391095161438, -22.97015380859375, 10.045456886291504, 12.35887336730957, 47.422271728515625, -36.3453369140625, -15.231244087219238, 13.422030448913574, -55.241878509521484, 7.6469645500183105, -81.44474792480469, -25.646913528442383, -18.094120025634766, 33.35409164428711, -4.780654430389404, -18.257129669189453, 77.47396850585938, 16.048864364624023, 56.012779235839844, 0.2382330447435379, 6.612498760223389, -19.527179718017578, -0.5277594923973083, 1.8526808023452759, 86.16471862792969, 10.068248748779297, -30.35566520690918, 4.650727272033691, 1.0785940885543823, -6.487380027770996, 20.927888870239258, 4.677511215209961, -5.748943328857422, 7.4424309730529785, -29.768463134765625, -13.101968765258789, 45.08648681640625, -31.757701873779297, -16.60662269592285, -71.138916015625, 3.049325942993164, 29.682048797607422, -35.007484436035156, -6.025959491729736, -5.887414932250977, 35.626590728759766, 17.158735275268555, 19.68151092529297, 30.340421676635742, 43.996315002441406, 12.307533264160156, -41.378089904785156, -6.195457935333252, -1.9996802806854248, -21.69879722595215, 14.093624114990234, -87.96988677978516, -37.43216323852539, -57.624507904052734, 33.597835540771484, -0.8545379638671875, -14.371635437011719, -17.645511627197266, -8.277512550354004, 14.519160270690918, -19.866111755371094, -8.866175651550293, 47.82841873168945, 15.960123062133789, 6.846242904663086, 47.03595733642578, 13.41094970703125, -8.556965827941895, 12.823381423950195, -25.163423538208008, 19.821889877319336, -26.7605037689209, 11.090763092041016, -8.836463928222656, -35.248504638671875, -46.35808181762695, 14.263547897338867, -54.711910247802734, -22.69451141357422, -21.553882598876953, -1.1183152198791504, 23.588830947875977, 9.943364143371582, 37.31521987915039, -11.320008277893066, -37.558467864990234, 7.071600914001465, -55.898006439208984, 20.597997665405273, 55.79106140136719, -24.06241226196289, -21.23204803466797, -26.456653594970703, -38.25145721435547, 3.4837634563446045, -7.545485019683838, -37.728816986083984, -14.694329261779785, -5.350235939025879, 9.889141082763672, -9.826452255249023, 29.169849395751953, -6.9965057373046875, -15.420602798461914, 19.20740509033203, -24.52362632751465, 58.560550689697266, -78.90120697021484, 28.657522201538086, -13.375890731811523, -47.83909606933594, 5.53418493270874, 34.1597785949707, 26.312599182128906, 72.8454818725586, -37.81705856323242, -20.924219131469727, 15.207756996154785, -35.15105438232422, -35.3120231628418, -25.885555267333984, -1.1346521377563477, -17.756135940551758, -13.874238014221191, 47.98092269897461, 13.838541984558105, 9.413660049438477, 32.88742446899414, 38.38706970214844, -1.4738645553588867, 2.8358545303344727, 27.760374069213867, 53.07500457763672, 27.6650333404541, -26.386106491088867, -27.56716537475586, 62.75533676147461, -2.382103681564331, 80.40504455566406, 13.691884994506836, 42.0189094543457, 62.19395446777344, 57.29631423950195, -3.246405839920044, -10.008642196655273, -1.3078179359436035, -19.342504501342773, 46.964942932128906, -44.966102600097656, 31.754955291748047, -18.184406280517578, -13.869094848632812, -47.58952713012695, -40.140655517578125, 11.496496200561523, 18.993133544921875, 0.3959612250328064, -7.397487163543701, 28.46354103088379, 15.975674629211426, -12.587299346923828, 32.826499938964844, 14.252884864807129, 11.092484474182129, -80.377197265625, -14.000031471252441, -6.63947057723999, -13.775032997131348, 10.23277473449707, -1.3093403577804565, 21.75790023803711, 51.047542572021484, 6.692418575286865, -12.732972145080566, -2.680527925491333, -14.662171363830566, 48.209678649902344, -74.31205749511719, -36.751243591308594, -6.417914867401123, 34.17815399169922, -11.482036590576172, 62.31258010864258, 53.098018646240234, 4.071395397186279, -12.177262306213379, -44.13056564331055, -14.528246879577637, -18.7429141998291, -41.73134231567383, -37.27702331542969, 15.695185661315918, -53.87128448486328, 14.1322021484375, 10.50575065612793, 1.1845698356628418, 9.969858169555664, 74.17491149902344, -23.607887268066406, -4.35867166519165, 16.593368530273438, 15.519463539123535, -28.74441146850586, -28.585927963256836, -50.32565689086914, -16.07328224182129, -14.557722091674805, 64.9129867553711, 11.314677238464355, 16.47125816345215, 1.0745348930358887, -57.175235748291016, 2.317197799682617, 23.38225746154785, 17.816986083984375, 16.311399459838867, 9.920547485351562, 86.04749298095703, 14.195462226867676, 10.436927795410156, -7.10905122756958, 24.00779151916504, -40.89149856567383, 25.144929885864258, 45.7387809753418, -45.31644821166992, 10.50723934173584, -14.612625122070312, -36.44426345825195, -6.565271854400635, -16.796857833862305, -42.435176849365234, 1.1393076181411743, -22.011030197143555, 2.663358449935913, -39.61088180541992, -37.93817138671875, 45.39094161987305, 7.419138431549072, -50.014583587646484, -14.267561912536621, 49.426761627197266, 44.8016471862793, -26.265106201171875, -16.94241714477539, -15.37964153289795, -56.126861572265625, 4.153357028961182, 9.64494514465332, 5.340301990509033, -92.26886749267578, 9.919289588928223, -23.86953353881836, -27.097631454467773, 19.890432357788086, -31.116588592529297, 43.01659393310547, -18.366668701171875, 1.2378671169281006, 39.66141128540039, -18.53510284423828, 9.05695629119873, 8.789164543151855, -0.15526136755943298, 19.206912994384766, -35.18135452270508, -18.115127563476562, -8.775753021240234, 14.842376708984375, -18.228397369384766, -24.877511978149414, 45.624481201171875, -26.345069885253906, 10.550402641296387, 22.376806259155273, -20.54033851623535, 33.006622314453125, -34.78034591674805, -41.966094970703125, 23.36467170715332, -4.248083591461182, 19.392240524291992, -4.691282272338867, 27.898265838623047, 52.64543533325195, 4.205088138580322, -24.3012638092041, 6.163783073425293, 23.09566879272461, -4.008764743804932, 78.25495147705078, 6.703512668609619, 17.855426788330078, -20.363014221191406, -1.3784857988357544, -25.283578872680664, -44.801513671875, 39.66119384765625, -39.479915618896484, -32.23367691040039, 8.2401704788208, -15.798433303833008, 13.372173309326172, -5.706508159637451, -40.23945999145508, -5.337693691253662, 11.993850708007812, -14.982897758483887, -0.8204318284988403, -32.073936462402344, 2.48335862159729, 36.6550178527832, -9.467001914978027, 13.23067855834961, -8.796077728271484, 14.456716537475586, -16.578588485717773, 7.863527774810791, -6.071743011474609, -0.9243435859680176, -36.52775192260742, 42.820072174072266, 34.44242858886719, 16.70286750793457, 54.21183776855469, -34.18476104736328, -8.324813842773438, 16.98126983642578, 3.68880033493042, 24.911787033081055, -6.06525993347168, 39.47523498535156, 32.0010986328125, 76.6414794921875, -25.27804183959961, -45.444828033447266, 32.826229095458984, 14.644601821899414, -11.869860649108887, -25.80097198486328, 8.993621826171875, 45.40715408325195, -55.408390045166016, -16.465255737304688, -20.6558837890625, 1.0848194360733032, 60.469364166259766, -12.695064544677734, -16.27066993713379, 2.9993338584899902, 30.909147262573242, -9.282618522644043, 34.77699661254883, 37.47662353515625, 46.50831604003906, -25.628799438476562, -22.967260360717773, -5.296823501586914, -88.93195343017578, -50.43842697143555, -39.29825973510742, -40.25459289550781, -19.225688934326172, 44.688663482666016, -28.90011978149414, 19.563997268676758, 11.759557723999023, -7.304853439331055, 62.81327819824219, -15.046902656555176, -8.94145679473877, 27.281789779663086, 4.478140830993652, -50.85163879394531, -10.620626449584961, -13.344082832336426, -33.53327941894531, 9.569746017456055, -10.389691352844238, -4.227834701538086, 15.23578929901123, -86.20748901367188, 3.4437036514282227, -2.4399192333221436, -5.72441291809082, 33.958003997802734, 52.49690628051758, -2.9892585277557373, 25.47953987121582, -33.22969055175781, -10.256412506103516, 46.76797866821289, -15.361085891723633, -40.214813232421875, -17.624652862548828, 65.63318634033203, -4.323463439941406, -73.05685424804688, 5.574828624725342, 11.676780700683594, -17.062044143676758, 12.034326553344727, 13.521137237548828, -29.388458251953125, 75.72687530517578, 11.743581771850586, 9.155755996704102, 24.756887435913086, 16.987590789794922, -26.39215850830078, 39.373226165771484, -0.2739417552947998, 8.695329666137695, -21.4121036529541, 18.301034927368164, 35.47077178955078, 11.286080360412598, 7.754454135894775, -41.71700668334961, -17.505905151367188, 48.81662368774414, 24.019977569580078, 13.603411674499512, -43.818119049072266, -43.5771484375, 36.90638732910156, -11.540529251098633, -24.188392639160156, 61.12766647338867, 20.06106185913086, 20.199234008789062, 27.08670997619629, 22.977191925048828, 21.051198959350586, 26.992469787597656, 19.075515747070312, 21.866107940673828, -21.696657180786133, 13.637340545654297, 45.6003303527832, -43.43217849731445, 22.654924392700195, 0.21817262470722198, 3.602438449859619, -17.3433780670166, 19.3293514251709, 19.991846084594727, 2.364649772644043, -22.36215591430664, 41.898094177246094, -31.643817901611328, -12.32514476776123, -36.62099838256836, -4.738478660583496, -11.154131889343262, 2.2128169536590576, -2.6390295028686523, 0.08310924470424652, -42.43046951293945, 34.2825927734375, -33.42103576660156, 15.858026504516602, -30.690622329711914, 28.812379837036133, -2.6074788570404053, 11.75648021697998, -7.082449436187744, 35.929603576660156, 7.258696556091309, -16.320377349853516, 16.413990020751953, 5.806844711303711, 27.325592041015625, 9.55597972869873, 1.4442404508590698, 7.653307914733887, 43.7336540222168, -7.391191482543945, -3.4081966876983643, -26.30043601989746, -40.904075622558594, -1.5889811515808105, 7.734241962432861, -11.00998306274414, -9.33867359161377, -60.83443832397461, 21.334335327148438, 33.56309127807617, -41.53860092163086, 4.595497131347656, -26.415496826171875, 8.625999450683594, -25.49188232421875, -26.243501663208008, -36.049991607666016, 46.05073928833008, -51.11777877807617, -42.49552536010742, -25.88247299194336, 48.82378387451172, -16.739091873168945, 28.43730926513672, -29.75020408630371, -2.0180137157440186, -33.95013427734375, -12.11002254486084, -51.64820861816406, -1.5252968072891235, -22.530675888061523, 30.543304443359375, 27.32882308959961, -38.554054260253906, 0.9102808833122253, 33.47612762451172, -4.068152904510498, 35.589752197265625, 63.255008697509766, -51.39618682861328, -16.237491607666016, -31.674270629882812, -41.41584396362305, -12.679373741149902, 18.615842819213867, 43.656982421875, 4.184097766876221, 6.1174421310424805, -26.950519561767578, 57.772361755371094, 44.34609603881836, -37.175514221191406, -11.844882011413574, 22.009849548339844, -0.27552586793899536, -52.135807037353516, -0.8524751663208008, -64.0030746459961, 5.533700942993164, -43.73122787475586, -14.770444869995117, 4.9159321784973145, -24.106908798217773, -4.280832290649414, 11.325066566467285, -8.26229476928711, -64.12149810791016, 27.451309204101562, 2.0328781604766846, 5.814311504364014, 7.262640953063965, -35.769840240478516, -11.200668334960938, -33.393253326416016, 10.896492958068848, -10.481740951538086, 20.50005531311035, -5.909920692443848, -1.836998701095581, 10.378865242004395, 26.533687591552734, 4.495781898498535, -37.003334045410156, 7.9053874015808105, -15.987902641296387, 25.726539611816406, 21.328075408935547, 28.762460708618164, 37.31337356567383, -7.406650066375732, -26.32929229736328, -16.55214500427246, -9.815186500549316, -41.313880920410156, -28.946779251098633, -8.178018569946289, 25.429473876953125, 34.783870697021484, -15.224136352539062, 24.102752685546875, -44.98861312866211, -21.403797149658203, -26.026504516601562, -20.110240936279297, 0.7588281631469727, 87.4914779663086, -6.022544860839844, 0.8245510458946228, -40.120609283447266, 6.868203639984131, 23.0111026763916, -88.74715423583984, 30.8375186920166, 3.295762777328491, 20.297061920166016, -35.12668228149414, -42.380775451660156, -17.322086334228516, 51.96793746948242, 1.6763633489608765, -27.565767288208008, 34.23298645019531, -8.45145034790039, 9.186904907226562, 37.22185516357422, 54.44763946533203, 18.936466217041016, -53.98811721801758, -26.37533187866211, -39.18825149536133, -36.23143005371094, -12.194225311279297, -67.1360855102539, 28.42544174194336, 25.642194747924805, -13.064215660095215, 35.552642822265625, 15.871868133544922, -25.1254940032959, 26.54279327392578, -13.629036903381348, -9.87501335144043, -31.49923324584961, -9.404437065124512, -50.64106750488281, -34.471405029296875, 26.070987701416016, 24.849613189697266, -44.95133972167969, -25.420368194580078, -17.2782039642334, -75.65442657470703, -9.719573974609375, -21.020978927612305, -6.266372203826904, 3.648189067840576, -29.089391708374023, 33.6222038269043, 11.026652336120605, -30.549072265625, -9.518927574157715, -3.5996646881103516, -16.933202743530273, 4.316971778869629, -11.495630264282227, -17.823020935058594, 47.436161041259766, 10.997225761413574, 28.22684669494629, 34.718162536621094, 61.778411865234375, -11.800419807434082, -0.5264322757720947, 8.504149436950684, -74.36705017089844, -17.686840057373047, 8.453344345092773, -38.748836517333984, -5.640620231628418, -53.71529006958008, -52.62314224243164, -21.4431209564209, 24.555156707763672, -0.26810282468795776, 22.824363708496094, 18.436275482177734, -21.98007583618164, 28.789073944091797, -39.1602897644043, 21.14742660522461, 68.49839782714844, -31.036720275878906, -4.293147087097168, -10.097525596618652, 35.66286087036133, -81.57098388671875, -21.083303451538086, 17.95043182373047, 47.77104568481445, -27.333282470703125, 42.191410064697266, -14.787141799926758, -36.07861328125, 10.630674362182617, -58.83787536621094, -43.69438171386719, -6.720607757568359, 14.759657859802246, -25.963010787963867, -5.934924602508545, -0.8447058796882629, 20.27186393737793, 1.3350838422775269, 37.658668518066406, -19.97756576538086, 1.6625488996505737, -21.135526657104492, -63.026546478271484, 41.22350311279297, -24.663660049438477, 47.1416015625, -26.797935485839844, 23.556644439697266, -3.683529853820801, -30.770118713378906, 44.24185562133789, 10.337677955627441, 33.437408447265625, -38.60431671142578, -54.87913513183594, -3.9577159881591797, 25.161523818969727, -2.4278767108917236, 25.458887100219727, -3.364813804626465, 6.735688209533691, -66.27465057373047, 27.35352897644043, 31.73297882080078, 4.136294841766357, 16.766983032226562, 30.62759017944336, -38.54861068725586, -25.843061447143555, 37.45891571044922, -3.675227165222168, -12.572834014892578, 21.001708984375, -54.83303451538086, 5.376481056213379, 28.398794174194336, -47.049739837646484, 10.322671890258789, -22.82809066772461, 6.051202774047852, 31.07549476623535, -20.24200439453125, 17.979848861694336, 2.278703451156616, -12.583788871765137, 61.9461669921875, -41.29121017456055, 30.884946823120117, 29.43705177307129, -32.90809631347656, 36.2646369934082, -17.423873901367188, 15.710674285888672, -35.792991638183594, 24.929407119750977, -40.14171600341797, -6.0586161613464355, 6.7891435623168945, 18.8465576171875, 29.54259490966797, 18.16249656677246, 59.909664154052734, 15.875564575195312, 11.731819152832031, 6.729279041290283, 51.4085693359375, 63.87518310546875, 26.556690216064453, -52.8841438293457, 65.77571105957031, 75.58007049560547, 35.41157531738281, -7.310266494750977, 0.417673796415329, 8.939387321472168, -1.3912540674209595, 26.74506187438965, 6.695921897888184, 12.033289909362793, -7.385530471801758, -54.62886428833008, -7.060119152069092, 31.820329666137695, 21.62615394592285, 25.39439582824707, 12.788962364196777, 2.0270965099334717, -19.462797164916992, -37.331295013427734, -20.88058853149414, 18.52399444580078, 50.74028015136719, -18.742551803588867, 44.924373626708984, 19.442325592041016, -45.10764694213867, -20.355819702148438, -36.35860824584961, -7.43185567855835, -8.468729972839355, -8.139156341552734, 53.79348373413086, 52.45603942871094, 8.500864028930664, 2.588669776916504, -50.097618103027344, 1.9665732383728027, -6.985650539398193, -18.229549407958984, -4.742708206176758, 53.630157470703125, 14.876238822937012, -34.5987434387207, 7.9848952293396, 16.252164840698242, -7.187205791473389, 16.79134750366211, -7.336455821990967, -16.019140243530273, 7.6117401123046875, -5.328007698059082, -20.330965042114258, -38.031558990478516, 8.874822616577148, 19.86577796936035, -29.165346145629883, -11.517281532287598, 47.94929504394531, 38.014801025390625, -3.235995292663574, 44.24081802368164, 3.4714410305023193, -12.176250457763672, 35.49014663696289, -3.1080422401428223, -16.62657928466797, -32.1265869140625, -22.91877555847168, 13.81977367401123, 22.47679328918457, -9.587538719177246, -0.9251829385757446, 19.856830596923828, 64.5220718383789, -73.38681030273438, -67.84593963623047, 57.68315505981445, -41.67721939086914, -59.68390655517578, -60.524131774902344, -52.47624206542969, -36.02431106567383, -4.1135711669921875, 59.974037170410156, -25.095561981201172, -4.003911018371582, 6.421616554260254, -69.23168182373047, -4.358091831207275, -75.30120086669922, -26.661741256713867, 37.96724319458008, -20.537609100341797, -22.392274856567383, -7.619219779968262, -12.340328216552734, -16.991239547729492, -24.822669982910156, 18.43524932861328, 71.81903839111328, -25.741697311401367, 9.24990463256836, -14.894020080566406, -10.349520683288574, -13.733353614807129, 18.230680465698242, -64.90898132324219, -43.290794372558594, 14.51182746887207, 35.30315399169922, -45.207332611083984, 8.048365592956543, -0.30029967427253723, 22.896442413330078, -29.097455978393555, 43.19612503051758, 4.689711570739746, -21.739770889282227, 45.41361618041992, 49.114234924316406, -39.833797454833984, 83.0831069946289, 59.70825958251953, 47.64518737792969, 9.6039400100708, -24.91074562072754, -15.353588104248047, -26.00296401977539, -51.772125244140625, -20.634876251220703, -51.231842041015625, -50.052146911621094, 14.649469375610352, 12.560405731201172, -17.293615341186523, -12.394436836242676, -7.035256385803223, -6.592284202575684, 39.00343704223633, -10.786168098449707, -10.40285873413086, 7.140134334564209, -33.816383361816406, 19.94126319885254, -8.525038719177246, -16.98417854309082, 2.1255664825439453, -9.044760704040527, -7.189302921295166, 16.970909118652344, -13.676854133605957, 23.755107879638672, -17.687084197998047, 34.28602600097656, 14.304693222045898, 11.480777740478516, 19.97504425048828, 40.57670593261719, -0.7123299837112427, -49.352054595947266, -2.74592661857605, 19.07914924621582, -25.546001434326172, 36.15938949584961, -21.502429962158203, -8.150291442871094, -15.570858001708984, 24.50201416015625, -10.948240280151367, 69.2564926147461, -35.97854995727539, 37.240318298339844, 31.331172943115234, -28.58896255493164, -52.3432731628418, -18.724599838256836, 9.421016693115234, 28.861385345458984, -22.392982482910156, -30.46665382385254, 15.527198791503906, -15.188606262207031, -21.749801635742188, 5.511536598205566, 29.745298385620117, 33.807373046875, -47.603187561035156, -24.895030975341797, 37.48860168457031, -25.860427856445312, 16.508499145507812, -35.8344841003418, -5.566267490386963, 40.434207916259766, 12.534420013427734, -0.42901667952537537, 6.572938919067383, 20.677217483520508, 1.5213526487350464, -5.417083263397217, -25.840333938598633, -16.623388290405273, -0.5968152284622192, 13.06854248046875, 12.506241798400879, 31.313291549682617, 5.826074123382568, -7.293742656707764, -21.58881950378418, 4.441927433013916, -51.96847152709961, 8.260038375854492, -16.55547332763672, -32.18510437011719, 30.480283737182617, 30.765806198120117, -32.9060173034668, -67.67254638671875, -30.464069366455078, -7.2928619384765625, -17.24688148498535, 2.3223936557769775, 45.935115814208984, -17.640071868896484, -44.577781677246094, 45.14305877685547, 24.650928497314453, -1.7724905014038086, 16.52650260925293, 4.553915500640869, -61.5426139831543, -14.459779739379883, -2.4049954414367676, 10.586893081665039, 26.009403228759766, -28.723339080810547, 8.44113826751709, 43.10393142700195, -58.154537200927734, -15.082996368408203, -1.911346673965454, -36.45220947265625, 38.00099182128906, 34.200828552246094, -16.25031280517578, 2.4113335609436035, -82.16806030273438, -25.92139434814453, -9.44011116027832, 3.7748730182647705, -37.14937210083008, -68.66342163085938, -81.95272064208984, 15.289830207824707, -16.758363723754883, 3.915020227432251, 27.75063705444336, 5.119627475738525, 12.144390106201172, -12.653203010559082, -27.097700119018555, -12.761228561401367, -3.5494043827056885, -19.793716430664062, -28.72223663330078, 46.20182418823242, -38.94622039794922, 44.80590057373047, -52.25360870361328, 5.730426788330078, 16.74494171142578, -4.584561347961426, -35.804473876953125, -22.764131546020508, -7.690539836883545, 3.8390793800354004, 36.52275466918945, 39.4581413269043, -21.7698917388916, 1.8747212886810303, -35.87413787841797, -11.820737838745117, 27.640363693237305, 10.39628791809082, 4.603323936462402, 47.14934158325195, -21.473962783813477, -17.18361473083496, -4.591882228851318, 9.506749153137207, 31.711650848388672, 17.377328872680664, -55.20822525024414, -5.5101847648620605, -22.953411102294922, -32.53422927856445, 40.306636810302734, -12.870006561279297, -15.025693893432617, -23.181804656982422, -51.690773010253906, 27.596397399902344, -37.648441314697266, -32.17555618286133, 14.228341102600098, -13.971583366394043, 2.408156156539917, 39.58807373046875, -33.412086486816406, 27.248531341552734, 19.430326461791992, 1.325492262840271, -17.753755569458008, -29.758319854736328, -33.74518585205078, -1.5497186183929443, 5.57462739944458, -55.177982330322266, 35.789405822753906, 21.881576538085938, -10.955530166625977, 64.16480255126953, -0.3555982708930969, -34.024208068847656, -56.6284065246582, -13.85210132598877, -8.192756652832031, -0.4852546453475952, 37.04314041137695, 59.39911651611328, -29.43330955505371, 8.928603172302246, -25.41014289855957, -11.05422592163086, 10.460050582885742, -13.67756175994873, -5.3134379386901855, 65.575927734375, -24.146602630615234, 3.1106069087982178, -25.864015579223633, 26.21756935119629, 35.46336364746094, 4.848306655883789, 44.78817367553711, -34.44163513183594, -28.704349517822266, -10.774595260620117, -47.613826751708984, 23.94116973876953, 6.920331954956055, 4.262508869171143, -28.906400680541992, 20.25316047668457, -12.954578399658203, -28.4493408203125, -3.976048469543457, -51.602901458740234, -17.278820037841797, 43.7201042175293, -1.8632874488830566, 17.531484603881836, 28.126955032348633, -7.161009311676025, -11.808050155639648, -37.39012145996094, -19.16840362548828, 32.674354553222656, 37.730552673339844, -11.750480651855469, -13.296916007995605, -24.301036834716797, -33.11021423339844, -33.328277587890625, 16.144472122192383, -13.839493751525879, -47.398414611816406, 3.7808141708374023, -81.54345703125, 48.43858337402344, 9.292167663574219, 47.83034133911133, -41.57184600830078, 20.376708984375, -3.883401870727539, 38.08598709106445, 5.404831409454346, 28.335771560668945, -6.502191066741943, -44.522090911865234, 17.926753997802734, 12.344671249389648, 19.953031539916992, 60.23529052734375, -7.006310939788818, -30.671689987182617, 6.584327220916748, 9.744132995605469, -0.833794116973877, 15.321701049804688, 62.63361358642578, 39.00160217285156, 38.74817657470703, 4.159191608428955, -43.57419204711914, 27.667583465576172, -27.984394073486328, 51.50155258178711, 49.467655181884766, -63.5009765625, 41.76935577392578, 20.66210174560547, -8.552169799804688, 16.354511260986328, 41.36086654663086, -22.64605140686035, -27.042579650878906, -41.81764221191406, 40.224674224853516, -61.39900588989258, -23.537939071655273, -43.722023010253906, -20.070119857788086, -24.987491607666016, 6.1851677894592285, -18.336673736572266, 81.62214660644531, 1.715335726737976, 12.500393867492676, -15.497342109680176, -26.63344955444336, -13.939767837524414, 1.682337760925293, 0.26951169967651367, -22.52936363220215, 5.8804402351379395, -18.998775482177734, -10.766302108764648, 18.44448471069336, 23.577648162841797, -24.48933982849121, 5.543396472930908, 2.532545804977417, -38.13813400268555, -12.63403606414795, 30.97161102294922, -17.25050163269043, -24.222362518310547, -23.889623641967773, -40.87062072753906, -24.34105110168457, 33.209632873535156, -17.624874114990234, 22.012128829956055, -22.38302993774414, -37.757598876953125, -9.782439231872559, 49.89929962158203, -12.92362117767334, -1.119513750076294, 40.65007400512695, -44.7700309753418, -15.645648956298828, 14.032705307006836, -6.457754611968994, 14.438021659851074, -8.196988105773926, -41.7769775390625, -23.67836570739746, -58.07723617553711, 38.98825454711914, -29.76233673095703, -8.187268257141113, -14.608668327331543, -41.62971878051758, 12.591484069824219, 55.603179931640625, -8.01819896697998, -70.31673431396484, -63.435001373291016, 5.515080451965332, 6.915822505950928, -5.263975620269775, -48.05900955200195, -38.08024978637695, 50.48798370361328, 33.55989456176758, -45.49201202392578, -37.283260345458984, 18.513622283935547, 29.96796417236328, -4.59830904006958, 7.5062575340271, 15.78223991394043, -0.2092778980731964, 6.717099189758301, -48.97956085205078, -2.162827491760254, -49.136356353759766, 30.634878158569336, -37.44204330444336, 7.192901134490967, -0.5513244271278381, -9.25882339477539, 38.344505310058594, -7.444268226623535, -30.696176528930664, -33.7331657409668, 20.51090431213379, 12.83715534210205, -22.46123695373535, 48.10919189453125, -7.917684555053711, -7.645008087158203, -11.458028793334961, -10.527040481567383, -12.720364570617676, -43.71888732910156, -49.37626647949219, -21.22491455078125, -7.094690799713135, 6.0878424644470215, -15.128725051879883, 19.278575897216797, 13.087417602539062, -37.513919830322266, -8.135009765625, -7.6558918952941895, 1.6195881366729736, -9.678447723388672, -23.6455020904541, 11.60806655883789, -45.78144836425781, 98.47151184082031, -37.165740966796875, 11.718156814575195, 22.220237731933594, -4.222465515136719, -50.110321044921875, 35.76526641845703, -20.97517204284668, 8.336984634399414, 15.71083927154541, -32.743778228759766, -23.848939895629883, 24.95428466796875, 41.71195983886719, -23.85249900817871, -5.993680000305176, 38.85863494873047, 0.44608938694000244, 80.22889709472656, 22.30596351623535, 31.235225677490234, -27.514963150024414, -24.436962127685547, 5.434968948364258, 17.935546875, -33.69731903076172, -4.929959297180176, -31.669553756713867, -2.17460560798645, -3.4668097496032715, 8.197053909301758, -3.9786217212677, -20.949731826782227, -28.91767692565918, 38.728641510009766, -2.7828922271728516, 9.602945327758789, 48.560638427734375, 0.8736979961395264, -37.44654083251953, 7.882001876831055, -30.84052848815918, 17.621774673461914, -15.024574279785156, -12.866812705993652, 11.596162796020508, -27.237468719482422, -72.3017578125, -64.14878845214844, 14.48326301574707, 16.460519790649414, 19.267658233642578, 7.110526084899902, 26.727392196655273, 42.92761993408203, -33.49557113647461, -23.741716384887695, 37.24119186401367, 17.46438980102539, 0.9842902421951294, 15.410317420959473, 12.570664405822754, -19.59282875061035, -10.093164443969727, 27.894617080688477, 13.234992980957031, -27.998676300048828, 13.532002449035645, 0.7016811370849609, -26.957687377929688, -36.56379699707031, 11.171368598937988, 21.444692611694336, -8.98370361328125, -27.77389907836914, 32.038055419921875, 27.63372802734375, -7.0609564781188965, -19.098922729492188, -3.5785906314849854, -30.514522552490234, 1.261782169342041, -24.56131935119629, -17.98749351501465, -36.61780548095703, -40.172569274902344, -33.84849166870117, 40.202552795410156, -32.65230178833008, 39.35569763183594, -31.002233505249023, -41.42888641357422, 4.7034502029418945, 22.328004837036133, 18.262121200561523, 24.919832229614258, -11.028168678283691, -29.778745651245117, -31.32745361328125, -61.63265609741211, 53.59566879272461, -26.52199363708496, 66.21794891357422, -19.77982521057129, -42.31591033935547, 5.8454790115356445, -32.989891052246094, -48.73551559448242, 28.063003540039062, -10.000874519348145, -24.948307037353516, -26.872398376464844, -19.33266258239746, 11.41357707977295, 15.127593994140625, -13.530802726745605, -22.45589828491211, -18.08758544921875, -9.90015697479248, 48.71125411987305, 42.17301940917969, 12.448564529418945, 20.305824279785156, 39.147216796875, -40.25953674316406, -30.75279998779297, -17.558956146240234, 22.03607177734375, 13.001741409301758, 32.31961441040039, -37.811553955078125, 20.462135314941406, -41.573509216308594, 32.778690338134766, -8.124800682067871, 5.6638994216918945, 11.493175506591797, -41.36941146850586, -15.03168773651123, 23.00733184814453, -32.08842849731445, -26.883577346801758, 19.394790649414062, -14.524604797363281, -8.930187225341797, -2.7857816219329834, 13.064230918884277, 0.5003331899642944, 56.102455139160156, 27.73577117919922, -8.11573600769043, -35.166473388671875, 13.936105728149414, -1.2372568845748901, -43.46579360961914, 13.86337661743164, 13.925447463989258, -9.524672508239746, 50.261749267578125, -23.457773208618164, 45.55289077758789, -3.6714839935302734, -1.645019292831421, -16.283077239990234, 21.79532241821289, 3.4056456089019775, 2.2555153369903564, -17.624879837036133, 1.4960243701934814, -2.070765733718872, 28.27501678466797, 4.24141788482666, 6.090466499328613, -8.486860275268555, -5.343799114227295, 12.538081169128418, 22.453174591064453, -15.705585479736328, 8.160628318786621, 21.610523223876953, 24.05215072631836, 3.9849486351013184, -23.821550369262695, 16.422801971435547, -3.96707820892334, 34.28313446044922, 6.772749423980713, 11.167673110961914, 29.829593658447266, -51.92939376831055, 31.03862953186035, 18.839534759521484, 9.191920280456543, 80.0547866821289, 24.99521255493164, -41.217063903808594, -0.10670238733291626, -6.0278639793396, -21.80157470703125, -17.437400817871094, 35.61244201660156, 0.26200494170188904, 24.43785285949707, -35.05284881591797, -7.6804680824279785, -20.210037231445312, 2.172734260559082, -16.17447280883789, -8.023683547973633, 11.077376365661621, 23.62213897705078, -45.16808319091797, 0.5017181634902954, 19.601112365722656, 1.039124846458435, 48.70131301879883, -18.16901397705078, -41.09920883178711, -36.20472717285156, 30.20523452758789, 7.164306640625, -33.997474670410156, 38.86177444458008, -1.4428081512451172, 12.772788047790527, 70.86604309082031, 27.07270050048828, 60.75218963623047, 11.2723970413208, 61.29901123046875, -40.956661224365234, -48.13765335083008, 21.06395149230957, -29.963993072509766, -23.638986587524414, -57.778076171875, -12.014528274536133, 43.112728118896484, -12.357283592224121, 44.16999435424805, -26.419384002685547, -51.18555450439453, -45.42380905151367, -13.680654525756836, -2.1114330291748047, 17.812646865844727, 29.714561462402344, -3.2954204082489014, 30.485679626464844, -1.6319890022277832, 10.110794067382812, -26.427967071533203, 14.714576721191406, 27.66179084777832, -20.407390594482422, 19.069026947021484, -8.088933944702148, 59.71726608276367, 0.7160186767578125, 37.28311538696289, 6.8089518547058105, 13.053947448730469, 35.59385299682617, 11.647123336791992, 39.6099967956543, 2.9027061462402344, 21.77536964416504, -20.51935386657715, -9.345022201538086, -0.6253169178962708, -13.19330883026123, 31.335100173950195, -3.4544837474823, 22.056360244750977, 56.695091247558594, 22.520797729492188, -1.0457210540771484, 9.508780479431152, 48.23192596435547, -4.310706615447998, 41.15214157104492, 46.535133361816406, -34.06700134277344, 19.596355438232422, 14.536935806274414, 17.832969665527344, -32.563846588134766, 5.324167251586914, -13.889616966247559, 7.691126823425293, 36.61498260498047, -22.044567108154297, -30.457551956176758, 58.79517364501953, -47.95186233520508, -14.269576072692871, 38.771400451660156, 20.9383544921875, 31.140514373779297, -6.392480373382568, 35.24509811401367, -73.35370635986328, -2.63417911529541, -13.86723804473877, -7.441860198974609, -11.895270347595215, -16.42853355407715, -53.24846267700195, -18.16614532470703, 35.099822998046875, -19.53779411315918, 13.34004020690918, -36.735469818115234, -9.863485336303711, 26.24650764465332, 26.00642204284668, -5.270484924316406, 8.784069061279297, 7.868566036224365, -25.45177459716797, 26.53728675842285, 9.261839866638184, 25.403156280517578, 17.738889694213867, 49.81890869140625, 14.076245307922363, -8.496774673461914, 17.766033172607422, 17.238224029541016, 3.9063634872436523, -8.807944297790527, -14.677931785583496, 21.353883743286133, -18.21843719482422, -15.366292953491211, -19.259151458740234, -31.99752426147461, -10.924596786499023, 40.79201126098633, -4.145356178283691, -18.1848201751709, -17.24747657775879, -50.20976638793945, -16.7264404296875, 3.418240785598755, -19.742334365844727, 7.386427879333496, 27.794851303100586, 14.00033187866211, -19.837209701538086, -18.015989303588867, 6.4518046379089355, 29.968534469604492, -61.045677185058594, -38.180885314941406, 38.36457443237305, 4.527831554412842, -23.642601013183594, -26.569557189941406, -27.652462005615234, 26.016878128051758, 29.03163719177246, -0.3043164312839508, 50.846839904785156, -19.053417205810547, -9.79218864440918, -35.88526916503906, -0.29688358306884766, 14.26477336883545, 54.96578598022461, 80.38130187988281, 16.495763778686523, -5.710419654846191, -18.68697166442871, -0.3772495985031128, 7.2879157066345215, 24.213743209838867, 7.657890796661377, 17.008548736572266, 8.78164005279541, 4.9479451179504395, 30.959693908691406, -4.543323993682861, -29.537731170654297, -22.526552200317383, -15.46882152557373, 71.55994415283203, -34.85908126831055, -18.205963134765625, -19.016542434692383, 59.65092468261719, 11.653376579284668, 31.335189819335938, -49.37376403808594, -6.020440101623535, 15.592052459716797, -8.354669570922852, 29.38758087158203, 54.97276306152344, -30.05556869506836, -25.883562088012695, 33.327171325683594, 37.28948211669922, 27.252925872802734, 38.77284240722656, -37.43093490600586, 17.582176208496094, -0.5320148468017578, 14.249393463134766, 12.042156219482422, 76.59180450439453, 62.382537841796875, 5.629826068878174, 48.27911376953125, 33.622257232666016, 24.97748565673828, 47.69980239868164, 54.19269943237305, -17.753225326538086, -19.95009994506836, -26.521631240844727, -14.856619834899902, -9.058769226074219, -2.038480758666992, 6.250954627990723, -13.089902877807617, 5.853414535522461, 15.288952827453613, 38.53788757324219, 32.63078689575195, 3.1002907752990723, -23.682579040527344, 71.75496673583984, -21.549585342407227, 8.138646125793457, -2.882674217224121, -8.722715377807617, -21.286222457885742, 20.972688674926758, -51.35131072998047, 39.80924606323242, 20.206371307373047, -29.432329177856445, -5.848442077636719, 7.010508060455322, 2.6388888359069824, 79.42522430419922, 32.467811584472656, 28.25981330871582, -7.915964126586914, 25.622756958007812, 2.607645034790039, 41.49785614013672, 8.961533546447754, 6.186208248138428, 20.228260040283203, 68.26566314697266, 22.59907341003418, -11.941658020019531, 40.5059928894043, -25.647735595703125, 46.78822326660156, 59.02207946777344, -15.854119300842285, 19.70733642578125, -36.347652435302734, 46.28233337402344, -12.767631530761719, -11.719084739685059, 10.082962036132812, 50.697471618652344, -30.93771743774414, -11.160984992980957, 40.37255096435547, -26.970436096191406, -9.322028160095215, -31.52644157409668, -12.445937156677246, 71.21495056152344, 53.264034271240234, 3.810607671737671, -2.1809282302856445, 57.160823822021484, -11.917919158935547, 23.65303611755371, 14.922722816467285, 4.541215419769287, 9.550695419311523, 54.1306266784668, -15.74050235748291, 37.77152633666992, 5.079051494598389, -43.884788513183594, 5.174062252044678, -12.090788841247559, 37.21732711791992, 29.19232749938965, -37.240665435791016, -29.133014678955078, 15.092239379882812, 38.19875717163086, -5.740882873535156, 6.8615288734436035, -2.582141399383545, 8.095454216003418, -5.506540298461914, -8.67623233795166, -70.19187927246094, -10.330915451049805, 14.823639869689941, -63.40953826904297, -1.215075135231018, -12.934027671813965, 8.270696640014648, 9.126379013061523, 30.90774917602539, 17.79340171813965, -6.312312126159668, 31.828847885131836, -0.9709187746047974, 40.11845397949219, -24.592803955078125, -21.041385650634766, -38.42219543457031, -23.621456146240234, -41.61268997192383, 18.89847755432129, -1.5920724868774414, -47.193382263183594, 29.524295806884766, 27.554981231689453, 25.42243766784668, -38.16238021850586, -25.8695068359375, -55.82093811035156, -20.32204246520996, -14.354750633239746, 21.418577194213867, 52.61189270019531, 30.518531799316406, -8.707175254821777, -4.75449800491333, -48.099578857421875, -10.9887056350708, 31.0429744720459, -21.81340789794922, 39.8349609375, 7.444465637207031, 60.508201599121094, 5.377560615539551, -15.429627418518066, -26.444622039794922, -10.390915870666504, -34.66014862060547, -48.733116149902344, -17.13069725036621, 4.595482349395752, -9.271324157714844, -7.378444194793701, 29.49532699584961, -17.325428009033203, -2.3434994220733643, -44.160369873046875, -9.533145904541016, -30.00946617126465, -35.68488311767578, -15.855245590209961, 2.3413772583007812, -33.88085174560547, 21.96564292907715, -0.08065568655729294, 18.324949264526367, 15.067509651184082, -66.06268310546875, 43.111793518066406, 39.077423095703125, -60.91790771484375, 85.27058410644531, 0.06755435466766357, -14.712493896484375, -42.121498107910156, -14.188541412353516, 7.6017656326293945, 38.97791290283203, -5.330167770385742, 21.78917121887207, -1.2337489128112793, -65.04255676269531, -56.42151641845703, 0.9190047979354858, 16.2882080078125, 20.422616958618164, 27.15924835205078, -23.072729110717773, 22.507740020751953, -4.823051452636719, -2.9122071266174316, 28.661544799804688, 13.008859634399414, -3.538926124572754, 34.16697311401367, 23.144489288330078, 21.372159957885742, 7.368578910827637, -30.421768188476562, 36.816871643066406, -17.531818389892578, -15.980953216552734, 21.415565490722656, 20.857316970825195, 42.56111526489258, 0.05840345099568367, -7.1993584632873535, 17.46864891052246, 40.93425369262695, -18.855300903320312, 2.982142686843872, 38.100914001464844, -4.225184917449951, -32.11189651489258, -29.11860466003418, 15.29669189453125, -36.271392822265625, 4.519181251525879, -7.574281215667725, 13.621975898742676, 8.969813346862793, -10.875144004821777, -0.6962897777557373, -36.34477615356445, -12.795564651489258, 57.8603401184082, -0.3626031279563904, 58.901004791259766, 29.206859588623047, 53.027198791503906, -70.30277252197266, 44.8551139831543, 22.917316436767578, -35.23585510253906, 29.415687561035156, -12.156230926513672, 65.52025604248047, 5.661628723144531, -4.33750057220459, 4.980349540710449, 5.580953121185303, 6.3500213623046875, -15.488927841186523, 1.099653720855713, 35.1734733581543, 11.659536361694336, 26.978397369384766, 11.061637878417969, -14.658787727355957, -65.99542999267578, -20.6914005279541, -24.80068016052246, -14.732645988464355, 6.683110237121582, 57.07673263549805, 8.963111877441406, 48.682960510253906, -28.70813751220703, -3.364741563796997, 12.929464340209961, 47.75518798828125, 2.628838539123535, 11.735405921936035, 14.689544677734375, 1.033933401107788, 2.6290042400360107, -5.615972995758057, -30.39160919189453, -4.784729957580566, -0.08039598912000656, 30.290010452270508, -37.73352813720703, 20.521034240722656, 16.33871078491211, -29.286035537719727, 27.959579467773438, 1.1727262735366821, 14.482481956481934, -30.679128646850586, -23.847944259643555, -9.698775291442871, 40.68563461303711, -30.904848098754883, -0.7830120921134949, 11.56824779510498, 57.65800857543945, -16.774246215820312, -33.05691909790039, -69.47270202636719, -2.523049831390381, -44.16850662231445, -16.201414108276367, -27.717679977416992, -6.474440097808838, 14.09418773651123, 13.605836868286133, -17.395177841186523, -2.364635467529297, 19.324581146240234, -54.796104431152344, 30.06353187561035, -49.658138275146484, 1.0520470142364502, 5.79234504699707, -20.5899658203125, 24.48027992248535, 10.744900703430176, -10.229144096374512, -48.500648498535156, 15.986092567443848, -7.8188652992248535, -4.77900505065918, 8.580787658691406, -54.40541076660156, -13.971263885498047, 16.394697189331055, -27.611377716064453, 33.662445068359375, 18.85341453552246, 14.27367877960205, 7.442641258239746, -1.1171027421951294, -10.650876998901367, 32.737022399902344, 5.140899181365967, 36.75848388671875, -52.34938430786133, -23.401052474975586, 57.57008361816406, -20.730201721191406, 4.473207950592041, 39.31585693359375, 78.94506072998047, -10.6852445602417, 30.766075134277344, -5.0318193435668945, -3.880556583404541, -11.326822280883789, 45.77626037597656, -28.257190704345703, -75.8559799194336, 12.701404571533203, -4.047407150268555, -10.843162536621094, -30.716293334960938, 9.456474304199219, 2.642650604248047, 4.177844524383545, -2.31009578704834, 65.24434661865234, 19.463048934936523, 59.912803649902344, 0.7712635397911072, -8.853574752807617, -14.509722709655762, 15.154586791992188, 44.55946731567383, 4.822473526000977, 15.22490119934082, -52.01840591430664, 21.592498779296875, -1.4039545059204102, 15.163911819458008, 18.7230167388916, -5.425497531890869, 2.6723175048828125, 20.907350540161133, -25.02681541442871, 23.823728561401367, -9.318881034851074, -10.617817878723145, 5.983865737915039, 31.873615264892578, -24.86098861694336, -9.29715347290039, 8.401939392089844, -12.657830238342285, 48.84393310546875, 42.77281951904297, 13.667107582092285, 7.509840965270996, 62.143985748291016, 36.84354019165039, -6.139482498168945, 15.496414184570312, 25.859779357910156, -24.40310287475586, -4.928759574890137, -9.897622108459473, -35.28776931762695, 5.8873186111450195, -65.79148864746094, 4.542574405670166, -11.932358741760254, -24.787437438964844, 12.663772583007812, -38.91094207763672, -5.1577467918396, -2.3076815605163574, 45.50476837158203, -16.737285614013672, 15.911911964416504, 57.439884185791016, 30.165616989135742, 16.13727569580078, 8.212634086608887, -22.439090728759766, -53.11846160888672, 19.469301223754883, -56.188472747802734, 9.025712013244629, -5.999527454376221, 24.020231246948242, -31.000789642333984, -39.505435943603516, 22.107269287109375, -8.68952751159668, 38.57737731933594, 0.10036678612232208, -22.200180053710938, 52.803138732910156, -3.3107147216796875, -5.774204254150391, 27.7971134185791, -39.19647979736328, -19.069936752319336, -27.118968963623047, -6.843439102172852, -34.177303314208984, -16.809301376342773, -32.04671096801758, 18.273052215576172, 10.900365829467773, -28.76622772216797, 13.373026847839355, 15.689329147338867, -22.079526901245117, 0.6192951202392578, 6.371662616729736, -14.135468482971191, 1.1185141801834106, -51.93522262573242, -52.56305694580078, 23.256391525268555, -32.576866149902344, 13.180069923400879, 30.696903228759766, 9.353686332702637, -12.18520736694336, -58.947105407714844, -17.1542911529541, -1.063057541847229, -24.556785583496094, 13.429457664489746, 6.455896377563477, -18.352781295776367, -18.523828506469727, -0.1884344518184662, -2.6161539554595947, -14.298965454101562, -19.055809020996094, 34.558780670166016, 48.08506774902344, -6.663161754608154, 20.771163940429688, -3.7903027534484863, 10.12702751159668, 42.053680419921875, 6.549320697784424, 33.54987716674805, 8.809748649597168, -42.02799987792969, -18.63111114501953, -46.505550384521484, -25.449195861816406, 24.95641326904297, -38.0796012878418, -15.423701286315918, 18.691843032836914, 23.089563369750977, 76.35184478759766, 4.954318523406982, -1.3661189079284668, -7.097579002380371, -19.536623001098633, 4.934147834777832, 38.622074127197266, -54.724021911621094, 5.545643329620361, 44.826324462890625, 22.655553817749023, 54.67507553100586, -12.195205688476562, 35.879051208496094, -36.2662467956543, 39.18939208984375, -32.81725311279297, 0.5913698673248291, 15.107746124267578, 20.28380012512207, 8.697784423828125, 32.491249084472656, 0.054417405277490616, -29.668140411376953, -55.330360412597656, 30.09650230407715, -6.35846471786499, -28.106182098388672, -1.8888417482376099, 53.57794952392578, -6.17454719543457, 8.771059036254883, 26.81346321105957, -40.979515075683594, -34.194801330566406, -12.258752822875977, -14.722262382507324, -29.083513259887695, 8.674155235290527, 1.932902216911316, 34.03992462158203, -34.18232727050781, 36.07720947265625, 6.282151222229004, 32.85590362548828, 35.715850830078125, 18.557260513305664, 46.30545425415039, -50.266841888427734, -27.538917541503906, -4.22050666809082, -5.85847806930542, 9.288283348083496, 29.263553619384766, -0.9550959467887878, 38.579742431640625, -65.92192840576172, 4.660969257354736, -1.0681118965148926, 37.78683090209961, -33.30686950683594, -2.6385080814361572, -7.064626216888428, 18.749561309814453, -11.863826751708984, -12.339345932006836, 21.169450759887695, -24.442230224609375, -6.287186145782471, 12.321783065795898, -12.013700485229492, 14.21728229522705, 57.54359436035156, -22.051090240478516, -30.349159240722656, -4.611577033996582, 19.623430252075195, 46.021785736083984, 0.31405481696128845, 18.614351272583008, -10.07272720336914, 7.0656304359436035, 24.26963996887207, -19.01348876953125, -31.095600128173828, 43.34164047241211, 39.27322006225586, 24.877416610717773, -3.957796096801758, -53.734561920166016, -44.42531967163086, 10.716381072998047, 7.756522178649902, -65.95218658447266, 10.993643760681152, 28.51225471496582, 2.4866960048675537, 26.277482986450195, -54.36686325073242, 33.42301559448242, 25.930133819580078, -6.404748439788818, 68.36666107177734, -8.42938232421875, -44.076229095458984, 10.931089401245117, -21.232227325439453, -16.276220321655273, -37.91337203979492, -34.76516342163086, 36.54828643798828, -10.233792304992676, 23.77141761779785, -30.95055389404297, -10.231951713562012, -14.446427345275879, -20.3604679107666, -2.401374340057373, -3.324998140335083, -19.140911102294922, 4.001265525817871, -3.935110330581665, -74.41671752929688, 43.401710510253906, 70.99276733398438, 13.093620300292969, 4.419018745422363, 60.30092239379883, 14.77161693572998, 9.878399848937988, 15.870881080627441, -20.573657989501953, 10.970194816589355, 13.000165939331055, -34.29542922973633, -8.059617042541504, 57.076324462890625, 10.786048889160156, 5.234081745147705, -19.42383575439453, 57.492820739746094, -4.733883380889893, -14.983028411865234, 15.176612854003906, 49.10142517089844, -10.649843215942383, -9.045042991638184, 12.16632080078125, 39.32048416137695, -26.07769775390625, 20.050596237182617, 26.743318557739258, -23.7305850982666, 13.579277992248535, 2.3334267139434814, 20.816919326782227, -30.529706954956055, -8.39390754699707, 2.188920021057129, 41.284767150878906, 4.9372172355651855, 13.159278869628906, 13.0538330078125, -0.28366151452064514, -2.3518459796905518, 26.24688720703125, -8.209102630615234, 75.81900787353516, -1.439436674118042, -16.60045623779297, 3.4427239894866943, -20.887819290161133, -32.783023834228516, -9.504006385803223, 2.5407495498657227, 1.4207017421722412, 84.6468734741211, -1.962083101272583, -19.61652946472168, 78.54440307617188, -9.429583549499512, -49.517127990722656, -29.03713607788086, 37.76163864135742, -3.070207118988037, 40.542076110839844, -14.575678825378418, -6.924289226531982, -9.532431602478027, -11.098276138305664, -23.102079391479492, 52.12034225463867, 32.672691345214844, 27.391372680664062, 31.753103256225586, 8.581755638122559, -6.683189392089844, 46.26403045654297, 32.282840728759766, -6.794095993041992, 17.331890106201172, 33.00395584106445, 1.5031325817108154, 77.02552032470703, 5.095301151275635, -23.4592342376709, 3.620384931564331, 5.75643253326416, -0.30427563190460205, 9.039910316467285, 35.59406661987305, 15.457304000854492, 3.82000732421875, 4.697266101837158, 25.15062141418457, 37.21956253051758, -3.2910523414611816, -26.828516006469727, -6.685362815856934, 36.40721130371094, -48.510616302490234, -20.675626754760742, -24.853281021118164, -34.86052322387695, -42.279197692871094, -24.58707618713379, 12.734306335449219, -27.204435348510742, 32.91645050048828, 13.049123764038086, 40.0596809387207, 26.5399169921875, 7.42210578918457, 26.263999938964844, 52.68000030517578, -44.82545852661133, 14.65412712097168, -49.53168487548828, 43.12445831298828, 11.334755897521973, -3.811403751373291, 47.491180419921875, 22.17112922668457, 35.56995391845703, 25.2540340423584, -9.49294662475586, -15.277006149291992, -11.033866882324219, 41.97530746459961, 7.2372541427612305, 40.01248550415039, -67.55671691894531, 7.93829345703125, -13.814526557922363, -18.529020309448242, 11.878229141235352, 23.89856719970703, -63.82965850830078, -4.624267578125, -22.193748474121094, -46.40182876586914, 38.34552001953125, 22.66124725341797, 19.348508834838867, -65.92192077636719, -41.09764099121094, -7.109333038330078, 34.86433029174805, -19.095497131347656, -30.674089431762695, -30.070201873779297, 55.88988494873047, 11.842567443847656, -19.44495964050293, 30.190290451049805, 37.15211486816406, -27.36505889892578, -51.082176208496094, -3.663240432739258, -70.1487808227539, -50.41850662231445, 9.379776000976562, 1.0184192657470703, 23.891643524169922, -24.415225982666016, -22.887954711914062, 21.535011291503906, -76.8121109008789, 23.511377334594727, 5.942291736602783, -11.720399856567383, 10.507779121398926, 13.922977447509766, 10.236251831054688, 20.85467529296875, 38.053977966308594, -54.95656204223633, -14.15850830078125, 20.018585205078125, 4.180098533630371, -20.991281509399414, -9.887288093566895, -10.569437980651855, 0.020485788583755493, -44.64470672607422, 28.569658279418945, 42.672882080078125, -1.90916907787323, -0.09765902906656265, -28.514957427978516, -47.29135513305664, 7.92672061920166, 13.020598411560059, 10.61610221862793, 5.01120662689209, -11.357378959655762, -33.19487380981445, 1.4851208925247192, 21.69630241394043, 33.531349182128906, -50.658382415771484, -22.357105255126953, 21.50649642944336, -36.92121887207031, -32.04278564453125, -16.765501022338867, -22.437946319580078, 6.656455039978027, -17.616060256958008, 53.759735107421875, 6.624380111694336, 23.961069107055664, 28.09069061279297, -2.610478162765503, -14.12181568145752, 19.12718963623047, -24.368114471435547, -30.65761375427246, -3.757305383682251, 31.868309020996094, 13.090807914733887, 14.527783393859863, -86.45341491699219, -33.46363830566406, -19.588741302490234, -19.546310424804688, -9.925313949584961, 45.21406173706055, 11.892179489135742, -1.6896765232086182, -10.549166679382324, -20.463958740234375, -29.607126235961914, -5.3493757247924805, -9.438990592956543, 16.648651123046875, 35.3056640625, 7.75929069519043, -66.27129364013672, 4.470834732055664, -33.5624885559082, -13.211542129516602, -25.611175537109375, -29.350818634033203, -11.506004333496094, 9.077019691467285, -12.020143508911133, -21.948436737060547, 22.405010223388672, 36.10824203491211, 39.54365158081055, 54.43172073364258, -2.0633463859558105, 51.39552307128906, -36.749755859375, 1.4300909042358398, 60.05253601074219, -24.881072998046875, -11.880729675292969, -17.348228454589844, 6.180693626403809, -19.018142700195312, -16.998653411865234, -36.493408203125, -54.73078918457031, 2.3389525413513184, 8.970369338989258, -23.819690704345703, -19.499496459960938, 21.267335891723633, 22.2309627532959, -28.48843002319336, -38.32898712158203, -18.598424911499023, 57.67825698852539, -21.838336944580078, 54.251617431640625, -33.2587890625], [-34.36459732055664, -8.462799072265625, 3.0288589000701904, 4.8377838134765625, -25.928659439086914, 2.8509232997894287, -23.53123664855957, -23.070571899414062, 29.97030258178711, -67.10063171386719, 20.289047241210938, -21.66343879699707, -28.914018630981445, 22.87184715270996, 13.930730819702148, 15.183320045471191, 19.076263427734375, -13.274395942687988, 32.99164581298828, 11.874773025512695, -2.3288350105285645, 8.702129364013672, 16.156686782836914, 45.361244201660156, 36.34682083129883, -24.09389877319336, 24.241493225097656, -1.781000018119812, 6.821900367736816, 7.16148042678833, -29.08135986328125, 25.789243698120117, -54.82086181640625, -5.679543495178223, 1.4688338041305542, -55.643558502197266, 17.09429931640625, 10.106207847595215, -12.839272499084473, -25.80449867248535, 18.753143310546875, 16.88673210144043, 13.292691230773926, 36.04118728637695, 2.9827630519866943, 23.789794921875, -11.646488189697266, 4.4739203453063965, 49.5179328918457, 41.801902770996094, 46.92601776123047, -0.6322781443595886, 28.768136978149414, -4.5658392906188965, 81.34619903564453, -28.821483612060547, -53.690128326416016, -25.74616813659668, -57.9345588684082, -62.20988464355469, -25.229028701782227, 0.7699865102767944, 53.91414260864258, -42.723331451416016, 14.118103981018066, -13.600364685058594, -12.067774772644043, 6.4868364334106445, -24.643150329589844, -42.69591522216797, -18.396533966064453, -16.18735122680664, -0.7518004775047302, 47.709354400634766, -17.793479919433594, -50.4276008605957, 0.3480510115623474, -39.536598205566406, -26.75393295288086, -75.36650848388672, -14.957296371459961, 18.126625061035156, -57.05907440185547, 67.08514404296875, -23.565567016601562, 62.036224365234375, -28.64971351623535, 9.113429069519043, -20.30849838256836, 28.255949020385742, 12.152580261230469, 36.43771743774414, 5.211915493011475, -4.645531177520752, -1.7675409317016602, 45.9876594543457, 4.174067974090576, 17.80900764465332, 5.511821746826172, 0.9379659295082092, -14.519414901733398, 41.76053237915039, -51.46416473388672, -19.728668212890625, 52.21341323852539, 11.879656791687012, -15.918609619140625, 13.214554786682129, -40.733741760253906, -31.226985931396484, 17.762765884399414, -37.730953216552734, 25.39552116394043, 45.62629318237305, -7.647359848022461, -31.101318359375, -58.688880920410156, 42.51836395263672, 24.910436630249023, -35.24212646484375, 17.450300216674805, -31.925979614257812, 53.40681457519531, 32.125755310058594, 11.39765739440918, 0.21348032355308533, -31.67691421508789, -12.540447235107422, 12.417363166809082, 1.2513870000839233, 8.5709228515625, -37.06655502319336, -13.658543586730957, 2.4365193843841553, 18.96988296508789, -44.578304290771484, 23.43002700805664, -48.320823669433594, 57.515933990478516, -14.631378173828125, 37.09513854980469, 7.432544231414795, 39.70476150512695, 26.003002166748047, -43.82509994506836, -40.494632720947266, 15.717963218688965, 21.95281410217285, 15.372666358947754, -2.949970245361328, 27.992250442504883, 17.255247116088867, 24.11918067932129, -4.405778408050537, 18.172754287719727, -45.0536994934082, 8.154982566833496, -35.43756103515625, 1.273301124572754, -19.730823516845703, -14.887953758239746, -30.655763626098633, -5.931825637817383, -20.124547958374023, 27.027254104614258, 40.85310363769531, 8.447196960449219, -24.77902603149414, 23.041290283203125, 39.212303161621094, 35.841835021972656, -27.624048233032227, -18.408546447753906, 19.841724395751953, -4.8488006591796875, 10.329080581665039, -21.920167922973633, -6.6892170906066895, -18.377174377441406, 8.868035316467285, -32.328792572021484, -53.85696792602539, 44.916378021240234, -66.20428466796875, -18.653709411621094, 55.446754455566406, 12.55339241027832, 52.62586212158203, -21.976016998291016, 4.11625337600708, -19.90669822692871, 16.972177505493164, 46.60395812988281, -26.459579467773438, -32.18000030517578, -17.867507934570312, -15.422186851501465, -1.5685043334960938, -7.265881538391113, -40.36117172241211, 40.090267181396484, -3.7163987159729004, -57.974708557128906, -30.184295654296875, -11.977625846862793, 35.95278549194336, 5.139364719390869, -54.81563949584961, 9.31190299987793, -12.365936279296875, 64.78573608398438, -39.517845153808594, -34.55940246582031, 24.3244571685791, 81.49539947509766, 44.38174819946289, 11.41443157196045, -27.35162353515625, -55.37247848510742, 9.051836967468262, 1.0394004583358765, 43.82877731323242, 31.787527084350586, 4.439288139343262, -50.5003776550293, -20.67169952392578, -7.61829948425293, -7.668718338012695, 61.953514099121094, 7.156508445739746, -78.58654022216797, 4.308315753936768, -15.310721397399902, 71.99066162109375, -24.238805770874023, 17.87055778503418, 1.5498543977737427, 0.6802690625190735, -28.232389450073242, -26.01114273071289, -2.2399191856384277, -24.86160659790039, 16.545333862304688, -0.174331396818161, -6.955132484436035, -29.29073715209961, 12.035463333129883, -26.623233795166016, -45.20405960083008, 31.195281982421875, -0.32079753279685974, 5.508057117462158, 57.96617126464844, 8.182415962219238, -4.062771797180176, -49.82228088378906, -25.177934646606445, 21.312297821044922, 58.81636428833008, -2.8124194145202637, 2.2330403327941895, 29.42169952392578, -11.176373481750488, -17.304595947265625, 36.01238250732422, -13.486516952514648, 17.267759323120117, 19.445186614990234, 29.910247802734375, 7.393803596496582, 58.88525390625, -3.9710121154785156, -43.087890625, 14.552977561950684, -8.3753662109375, 19.800151824951172, 31.927417755126953, 20.014049530029297, -20.425029754638672, -44.27742004394531, 2.2119593620300293, -39.54407501220703, -28.81545066833496, 50.52260208129883, -40.547767639160156, 17.808942794799805, 2.0830891132354736, 21.527956008911133, 23.78629493713379, -30.72816276550293, 5.959417343139648, -7.180313587188721, 32.78668212890625, 37.18867111206055, 53.761810302734375, 12.214386940002441, -7.958746910095215, 16.028282165527344, -37.225196838378906, -39.6436767578125, 12.903558731079102, -62.51442337036133, -17.076751708984375, 45.15803909301758, 24.021631240844727, 85.14832305908203, 52.24814224243164, 5.640695095062256, 18.399051666259766, 60.19631576538086, -5.14132833480835, 73.63799285888672, -34.81166076660156, 54.35136032104492, 31.338054656982422, 49.85392379760742, 33.915504455566406, 12.686285018920898, -1.975562334060669, 7.252342224121094, 3.3711721897125244, 13.264253616333008, 33.66637420654297, 5.6298699378967285, -25.773311614990234, -26.786500930786133, -22.534400939941406, -41.873172760009766, -57.66388702392578, -49.16361999511719, 43.648597717285156, -15.087130546569824, -15.291650772094727, 19.947776794433594, -26.559789657592773, -3.8967745304107666, 25.299911499023438, -17.48484992980957, -40.40494155883789, 5.594991207122803, -4.307094097137451, 9.229108810424805, 16.674348831176758, -37.491844177246094, 64.07746887207031, 27.807090759277344, -10.596437454223633, 13.017425537109375, 45.330078125, -19.213695526123047, 7.531130790710449, 0.2835252285003662, -73.8149185180664, 12.454060554504395, 26.265932083129883, -24.612138748168945, -40.82911682128906, -37.096160888671875, -27.577655792236328, -23.030527114868164, -19.21051025390625, 1.1273112297058105, -60.270408630371094, 85.77352905273438, -37.35935974121094, -11.451250076293945, 12.684475898742676, -33.108131408691406, -5.49357271194458, 35.90205001831055, -19.738096237182617, -39.45731735229492, -51.4948616027832, -46.34262466430664, 7.823648929595947, -14.039961814880371, 17.98683738708496, 56.63037109375, -18.806127548217773, 1.0890905857086182, -35.6289176940918, 47.72762680053711, -3.634038209915161, 5.5273284912109375, 33.48825454711914, 74.44999694824219, 70.79439544677734, -28.20423126220703, -62.24601745605469, 2.9785287380218506, -30.812040328979492, 44.572818756103516, 43.82548522949219, 8.690430641174316, 1.6473050117492676, -32.160804748535156, 26.653488159179688, 13.292882919311523, 16.504587173461914, -20.431503295898438, -54.202796936035156, -9.709379196166992, -48.474830627441406, -1.928653597831726, 11.832673072814941, 3.955714702606201, 26.408527374267578, -4.6171722412109375, -44.46807861328125, 15.287114143371582, 12.859124183654785, 16.175521850585938, 27.63136863708496, 19.549026489257812, 48.35984420776367, -28.218196868896484, -35.590457916259766, -43.54872512817383, 15.666183471679688, -26.059188842773438, -40.93434143066406, 40.82493209838867, 30.359949111938477, -31.833446502685547, -7.140976905822754, 44.15780258178711, 36.827571868896484, 38.600746154785156, -7.961714744567871, -48.02128219604492, 1.1665047407150269, -55.483699798583984, -11.326760292053223, -35.75068283081055, -9.702796936035156, -12.672399520874023, 1.2122135162353516, 15.955116271972656, 2.166038751602173, -8.526320457458496, 67.0313720703125, -12.014429092407227, -27.354400634765625, 19.574356079101562, 3.4851784706115723, 49.719390869140625, -53.6873779296875, -9.256925582885742, 4.713558673858643, -13.018142700195312, 33.54977798461914, -2.353822708129883, 1.5011178255081177, -18.1015625, 57.163612365722656, 13.849616050720215, -30.065914154052734, -19.247026443481445, -28.339622497558594, -1.074284315109253, -15.23633861541748, 10.15959644317627, -8.756170272827148, 27.684558868408203, -13.577776908874512, -3.0329055786132812, -5.111607074737549, -1.439919114112854, -18.05200958251953, -7.542367458343506, 31.952354431152344, -61.111385345458984, 6.956295013427734, 13.693050384521484, -24.11447525024414, -57.71096420288086, 38.176597595214844, -39.19771194458008, 4.9241862297058105, 14.618932723999023, -25.7869873046875, 27.629257202148438, 24.31699562072754, -24.96198844909668, 13.601739883422852, -0.7673919796943665, -1.5180511474609375, 26.345645904541016, 21.508590698242188, 11.409451484680176, 23.45172882080078, -29.208852767944336, -35.9930419921875, 29.56055450439453, 1.704300045967102, -24.857851028442383, -13.380950927734375, -18.09237289428711, 12.590218544006348, -34.03641128540039, 20.67542839050293, -5.109551906585693, -5.628718376159668, 11.615216255187988, 29.03609275817871, 41.09620666503906, 33.1690673828125, -20.946565628051758, -16.42664337158203, 39.3775749206543, -26.88043212890625, -6.936047554016113, -26.656688690185547, -10.05955696105957, 0.5191267132759094, -26.686925888061523, -40.729244232177734, -11.040459632873535, -54.77076721191406, -57.23596954345703, -12.04628849029541, 34.128231048583984, 29.30379867553711, -23.62480354309082, -3.4639499187469482, 0.2843601107597351, -2.8070714473724365, -0.9673724174499512, -23.22840690612793, 13.35135269165039, 29.360916137695312, 21.529882431030273, 48.57329177856445, -32.83318328857422, -13.55703067779541, 8.585136413574219, 20.150575637817383, 9.650382995605469, 35.45516586303711, 24.734052658081055, -17.158395767211914, -60.76567459106445, 24.678316116333008, 2.6484663486480713, -18.067604064941406, -28.35472869873047, -12.165831565856934, -19.569244384765625, -13.382689476013184, -46.41458511352539, 33.320743560791016, 30.760892868041992, 38.60189437866211, -57.63217544555664, -19.378639221191406, 23.111528396606445, 17.653411865234375, -8.844324111938477, 26.34197235107422, -13.332996368408203, 11.969099998474121, -18.971019744873047, -0.5344327092170715, -0.3126339912414551, -79.3126220703125, 65.98640441894531, -4.737245559692383, -58.69160842895508, 37.5589485168457, 1.683327317237854, 41.48431396484375, -46.79513168334961, -34.843624114990234, 9.96280288696289, -28.35553550720215, -14.799589157104492, 18.802865982055664, 6.546211242675781, -5.742852210998535, 13.003328323364258, -6.495560169219971, -18.59124183654785, 43.012535095214844, -42.65129089355469, 75.56285095214844, 32.23149871826172, -11.922770500183105, 5.499690055847168, -9.633085250854492, -16.477310180664062, 26.771142959594727, -78.96458435058594, 10.45590877532959, -35.276878356933594, -20.581003189086914, 24.74781608581543, -9.324440002441406, -14.192429542541504, -87.84904479980469, -42.53253173828125, 14.848786354064941, -62.12220764160156, -60.08512878417969, -12.613279342651367, 24.201295852661133, 32.69742202758789, -37.362464904785156, 26.57112693786621, -16.132946014404297, 11.958471298217773, -28.574487686157227, -48.37106704711914, 13.437962532043457, -7.451263904571533, -12.247339248657227, -28.1616153717041, 57.74182891845703, 19.148103713989258, -13.557948112487793, 53.53441619873047, -24.96540069580078, 17.13529396057129, -10.037881851196289, -5.943967819213867, -20.298803329467773, 4.795323848724365, -24.987028121948242, -1.7802958488464355, -0.047936636954545975, 83.17219543457031, -53.868858337402344, -29.620832443237305, 31.824180603027344, -6.975149154663086, -19.365694046020508, 8.232429504394531, 7.398046493530273, -29.011505126953125, -6.116538047790527, 32.413787841796875, -18.8450984954834, 17.19948959350586, -7.226197719573975, 5.952321529388428, 14.560026168823242, -9.23766803741455, -6.366240978240967, -15.357268333435059, 11.929044723510742, -9.418526649475098, 10.389339447021484, 11.672757148742676, 8.395393371582031, -0.8407613039016724, 35.056121826171875, 12.12382984161377, 4.13565731048584, -38.79303741455078, -42.321998596191406, 3.905708074569702, -35.86854553222656, 23.955116271972656, 17.157005310058594, -1.9950569868087769, -1.7438768148422241, 3.549194097518921, -22.601797103881836, 4.962351322174072, 9.095111846923828, 43.77412033081055, 8.918118476867676, 11.694705963134766, 39.51279067993164, -47.52836990356445, -32.60372543334961, 20.19194221496582, 60.795799255371094, 20.941787719726562, 2.050347328186035, 20.19803810119629, -18.085899353027344, 22.626955032348633, -22.11849021911621, -16.06245994567871, -36.072731018066406, 2.0373570919036865, 7.886857032775879, 34.2073860168457, -15.131854057312012, 23.376977920532227, -33.5435676574707, -22.486270904541016, -16.832860946655273, 17.260461807250977, 5.811502456665039, 9.918456077575684, -14.301149368286133, -19.23078727722168, -0.044645726680755615, 17.877370834350586, -27.287338256835938, -29.862377166748047, 5.703495502471924, 45.677852630615234, -48.457149505615234, -12.551140785217285, 37.93918228149414, -16.24575424194336, 8.349906921386719, 22.698604583740234, -50.14738845825195, -0.6704909801483154, 17.560352325439453, 49.794212341308594, 38.361122131347656, 14.185755729675293, -1.0217413902282715, 9.6638822555542, 28.440166473388672, 49.854515075683594, 20.999332427978516, 27.308055877685547, -26.900663375854492, -18.465744018554688, -39.44306564331055, 65.35460662841797, -2.2389283180236816, -43.99280548095703, 1.2905141115188599, -12.42018985748291, 33.5109748840332, -4.201941967010498, -11.792719841003418, 1.063739538192749, -47.35544967651367, 7.714393615722656, -54.99068832397461, 5.644436836242676, 68.89456939697266, -6.069174289703369, 40.89161682128906, 0.8128679990768433, 44.79508590698242, -5.47264289855957, -7.965324878692627, 31.324951171875, -27.815549850463867, 4.968786239624023, -18.465059280395508, 42.925228118896484, 65.64147186279297, 29.179372787475586, 40.964656829833984, 34.84244155883789, -1.4378412961959839, -17.768367767333984, -24.111297607421875, -46.50783920288086, 16.849504470825195, 14.529020309448242, -21.894365310668945, -0.6740102767944336, 2.632164716720581, 5.276500225067139, 2.5457077026367188, -13.732730865478516, -31.530929565429688, -65.30889892578125, -33.451133728027344, -11.060003280639648, 66.02803039550781, -50.3698616027832, -14.492803573608398, 39.792232513427734, -43.57013702392578, 9.081892013549805, -44.435333251953125, -3.5915613174438477, 11.179227828979492, -20.101041793823242, -8.797491073608398, 9.261449813842773, -10.368426322937012, -3.4072747230529785, 7.144789695739746, -14.757336616516113, -6.939053535461426, 16.480274200439453, -27.351348876953125, 10.744693756103516, -32.45113754272461, 5.72222375869751, 13.43024730682373, 16.17942237854004, 12.02003288269043, -12.965539932250977, -19.99069595336914, -70.24198150634766, 39.78166198730469, 9.173567771911621, 5.661159515380859, -16.12051010131836, -43.98063278198242, -3.3205482959747314, 22.11716651916504, 46.933570861816406, 15.084285736083984, 33.24040985107422, 9.474617004394531, 24.62506675720215, 41.51808166503906, 28.889236450195312, -39.538665771484375, 2.2757017612457275, -6.472301959991455, -28.464603424072266, -15.177240371704102, 7.332325458526611, -11.391561508178711, -14.256673812866211, -4.666965007781982, 31.189598083496094, -14.249969482421875, -32.70149612426758, 23.343645095825195, 4.122148036956787, 21.640615463256836, -0.5489858388900757, -62.4192008972168, -27.86397361755371, 35.00899124145508, 22.875547409057617, 39.646785736083984, 73.23944091796875, -2.6105165481567383, 45.74135208129883, 1.820530652999878, -13.671045303344727, -7.74445104598999, 15.362597465515137, 55.169288635253906, -61.16566467285156, 37.96760940551758, 9.210143089294434, -39.217994689941406, -31.140954971313477, 13.914552688598633, -67.4388427734375, 55.74724197387695, 34.120361328125, 44.27545928955078, -35.34973907470703, -21.078128814697266, 0.08077912777662277, 63.792137145996094, -19.772144317626953, -19.94723892211914, 46.174522399902344, -32.529048919677734, -21.48108673095703, -8.729615211486816, 30.450002670288086, -29.04129981994629, 22.98151397705078, -15.534475326538086, 21.374061584472656, 0.38397473096847534, -11.946640968322754, -11.51505184173584, 30.071542739868164, -12.745253562927246, -14.562519073486328, -19.398897171020508, 8.677265167236328, -52.794464111328125, 18.429941177368164, -11.673345565795898, 3.8657805919647217, -32.4665412902832, 24.188318252563477, -33.75242233276367, -0.6863785982131958, 19.956586837768555, 33.38318634033203, 4.944347858428955, 29.467350006103516, 23.01092529296875, 30.43868064880371, 4.215185165405273, 22.79824447631836, -46.216224670410156, -20.225170135498047, -4.1853766441345215, 14.364554405212402, -29.507232666015625, -13.978273391723633, 36.27408218383789, -24.094009399414062, 7.940796852111816, -37.118778228759766, 47.921592712402344, 12.722490310668945, -47.26750564575195, 0.6940804719924927, 37.82280731201172, -8.76392936706543, 3.5778441429138184, 14.209137916564941, -19.89348793029785, 11.98602294921875, 11.553109169006348, -22.179378509521484, -26.931903839111328, 15.25744342803955, -2.6964614391326904, 44.93973159790039, -22.61518669128418, 17.71843719482422, -39.4984245300293, -15.499456405639648, 1.4992594718933105, 5.013733386993408, -10.33403491973877, 32.860206604003906, -32.14374923706055, 20.864097595214844, 21.903728485107422, 57.172122955322266, -1.280178427696228, 47.29745101928711, -39.338714599609375, -36.94459915161133, -1.2870405912399292, -31.639680862426758, -6.277005672454834, -32.105262756347656, -42.642486572265625, -31.630138397216797, -50.87921905517578, -0.5145624876022339, -0.7045725584030151, 50.864723205566406, 50.46932601928711, 7.952246189117432, -39.752986907958984, -51.876609802246094, 9.361841201782227, -0.240128755569458, 36.09641647338867, -12.080931663513184, -14.161115646362305, 37.069602966308594, -9.679908752441406, 12.002110481262207, -33.831260681152344, 55.28493118286133, 9.851810455322266, -44.839637756347656, 7.146084308624268, 33.47584533691406, 18.813560485839844, 13.872122764587402, 30.82076644897461, -16.171531677246094, -14.24227237701416, -56.04451370239258, 3.242095470428467, -26.36829376220703, 32.662750244140625, -3.706139087677002, 32.74256134033203, 2.4365358352661133, -1.3682740926742554, 14.261434555053711, 27.93031120300293, 30.40424156188965, 17.261186599731445, -7.798799991607666, 2.27545166015625, 3.1606669425964355, -15.581106185913086, -13.342366218566895, -21.767301559448242, -34.88304901123047, -4.368854522705078, 42.32207107543945, -5.254249572753906, -1.8781272172927856, -25.831310272216797, -21.70063591003418, -17.480117797851562, 21.128877639770508, 6.41198205947876, 42.8956298828125, 4.941164970397949, 23.78685760498047, -42.89919662475586, 29.873470306396484, -37.03833770751953, 5.710240364074707, -36.17201232910156, -27.588726043701172, -28.674806594848633, -28.2353515625, 72.28871154785156, -2.2013535499572754, -2.0141754150390625, -7.469410419464111, -15.922018051147461, -64.2176742553711, 17.541545867919922, 59.10219955444336, 9.676257133483887, -28.558162689208984, -24.477827072143555, -26.728076934814453, 19.05586051940918, -33.941871643066406, -13.91501235961914, 1.0383813381195068, -3.635241985321045, 57.65278625488281, -57.75947952270508, -15.843679428100586, -35.473289489746094, -17.381006240844727, -8.19763469696045, -13.550387382507324, 7.916664123535156, -7.207760810852051, 31.05146598815918, -20.77696418762207, -1.6492974758148193, 30.42757225036621, -30.534557342529297, -5.942536354064941, -15.815886497497559, 29.451236724853516, -9.030855178833008, -18.05093765258789, 49.19039535522461, -23.765649795532227, 1.6969202756881714, -60.449832916259766, 10.706720352172852, 0.6132866740226746, 95.41724395751953, -28.552410125732422, 36.9370231628418, 1.9877090454101562, 17.168298721313477, -15.137321472167969, 29.765426635742188, 32.300594329833984, -30.405990600585938, -58.37371826171875, 29.297361373901367, 13.221811294555664, -4.53800106048584, 7.302527904510498, -45.6432991027832, -31.20267677307129, 38.40050506591797, -12.795498847961426, -36.926727294921875, -26.882692337036133, -6.077793598175049, -1.3671551942825317, 12.77479076385498, 64.12361907958984, 28.843116760253906, -0.10428422689437866, 0.5497025847434998, -3.793388843536377, -33.632781982421875, -6.757690906524658, 60.0443229675293, -5.5482354164123535, -65.2394027709961, -11.527921676635742, 11.385380744934082, 37.46906280517578, 12.507521629333496, 12.478982925415039, 38.69338607788086, 3.943319320678711, 20.59135627746582, 7.281708717346191, 2.14395809173584, -39.37181854248047, -2.788652181625366, 5.593356132507324, -7.674291610717773, 20.827213287353516, -25.956314086914062, -13.18725299835205, -50.1710319519043, 33.05071258544922, 7.23253870010376, 50.58340072631836, 11.967957496643066, -19.67267417907715, 14.812873840332031, 24.58775520324707, -26.036884307861328, 28.772018432617188, -30.96599578857422, 33.81642532348633, -23.898712158203125, 2.225198745727539, 45.39464569091797, 2.8346402645111084, 20.164249420166016, 6.515547752380371, 28.067760467529297, 33.48280334472656, -24.49529457092285, -19.176807403564453, 18.57684898376465, -44.408260345458984, -35.02659606933594, -47.88182067871094, -46.28059387207031, 1.225105881690979, 29.46819496154785, -30.32925796508789, 38.41556930541992, 11.410507202148438, 35.115081787109375, -4.3530354499816895, 24.964643478393555, 52.4732551574707, -18.26175880432129, -19.320938110351562, -23.145856857299805, 5.631355285644531, 14.231382369995117, 2.2847888469696045, 53.724700927734375, -18.172260284423828, 25.978031158447266, 21.315521240234375, -14.804376602172852, -37.86008834838867, 30.685693740844727, 47.420204162597656, 48.153804779052734, 11.624115943908691, -27.41904640197754, -6.8949198722839355, 8.696131706237793, -63.44721221923828, -9.580955505371094, 69.24335479736328, 18.83389663696289, 34.1939582824707, -23.68821144104004, -3.263300895690918, 12.925561904907227, 24.575363159179688, 65.29212188720703, 74.08168029785156, -17.718467712402344, 40.03276062011719, 13.538192749023438, -26.74643898010254, 38.36875915527344, 10.418021202087402, 39.729190826416016, 0.612236499786377, -16.843856811523438, -2.028834104537964, 64.44540405273438, -4.5835490226745605, -25.79753875732422, -4.366330146789551, 32.898338317871094, 6.7704176902771, -45.55216979980469, -2.7054598331451416, 19.48666000366211, 2.337096929550171, 23.346681594848633, -14.514045715332031, 20.25385093688965, 15.928784370422363, 12.858073234558105, -6.678351879119873, 65.6782455444336, -12.035213470458984, 5.814451217651367, -13.890693664550781, -4.09615421295166, -70.02328491210938, 12.239816665649414, 0.7518446445465088, 13.634747505187988, -12.687016487121582, 24.014863967895508, -23.92185401916504, -16.2071475982666, 21.269577026367188, -48.60209274291992, -29.834115982055664, -44.7795524597168, 16.419063568115234, -16.908119201660156, 22.360950469970703, 6.599358558654785, 32.87262725830078, 0.2673712968826294, 6.216528415679932, -23.530092239379883, -36.577980041503906, -0.9230918884277344, -34.57732009887695, 5.245456218719482, -31.81266212463379, 9.06971549987793, 30.116558074951172, -37.68198776245117, -11.547367095947266, 12.56640338897705, -4.198506832122803, -21.745460510253906, 21.112396240234375, -4.512029647827148, 38.39089584350586, -33.817413330078125, -52.095157623291016, -7.287070274353027, 3.72184681892395, 20.510478973388672, 31.52403450012207, 29.692432403564453, 62.98255157470703, 13.292877197265625, -0.40355679392814636, 12.367731094360352, -6.26353645324707, 39.7918815612793, -6.847479820251465, -27.348007202148438, -5.169261455535889, -20.239025115966797, -5.828509330749512, -40.028778076171875, 31.058815002441406, 18.164106369018555, -15.577718734741211, 2.3727986812591553, 42.53974151611328, 10.770072937011719, 6.667681694030762, -8.976048469543457, -39.41371154785156, -13.496694564819336, 59.107574462890625, -11.001249313354492, -9.657245635986328, -13.291525840759277, 19.180980682373047, -76.05484771728516, -22.831493377685547, -28.07990264892578, -39.285369873046875, 18.640151977539062, -16.328508377075195, 10.862886428833008, 27.839092254638672, -16.268983840942383, 6.942679405212402, 32.67644119262695, -2.6027705669403076, 22.79407501220703, -31.828533172607422, -9.170540809631348, -29.26238250732422, -1.2167319059371948, -50.932586669921875, -5.455110549926758, -33.11332321166992, -28.246410369873047, 46.769569396972656, -14.287226676940918, -3.394874095916748, -28.125513076782227, -62.26405715942383, 16.213502883911133, -17.508563995361328, 1.2726980447769165, 42.61639404296875, -13.893104553222656, -3.6450953483581543, 6.532636642456055, -21.356689453125, -28.562116622924805, -15.263036727905273, -0.1087828204035759, -39.021121978759766, -38.00716018676758, 12.374763488769531, 9.989110946655273, -12.918473243713379, -15.575899124145508, -3.210432291030884, 11.612656593322754, 35.14170455932617, -7.524251461029053, 15.606377601623535, 5.7799391746521, -1.03618323802948, -45.386898040771484, -4.955728530883789, 38.06919479370117, -30.641220092773438, 2.385256767272949, 45.99759292602539, -0.1662568598985672, -87.91474151611328, 45.49464416503906, -72.3973159790039, -20.500566482543945, -12.969273567199707, -24.58905601501465, -5.1026740074157715, 11.460707664489746, 4.560640811920166, -62.95871353149414, 8.204877853393555, -28.399951934814453, 18.472890853881836, 42.67761993408203, -37.66172409057617, -28.162809371948242, -15.64510726928711, -48.83717346191406, -7.762543678283691, -42.765533447265625, 58.07673263549805, 5.431812286376953, 70.98441314697266, -36.184532165527344, -45.68379592895508, 26.127553939819336, -5.040065288543701, 45.201072692871094, 43.38080596923828, 21.97867774963379, -32.07260513305664, -4.472867965698242, 16.556711196899414, 8.066798210144043, 18.789857864379883, 40.12441635131836, 10.16475772857666, -63.18002700805664, -16.558752059936523, 14.734404563903809, 19.21923828125, 78.0916519165039, 49.35251235961914, -60.834129333496094, -4.05285120010376, 22.053831100463867, 17.395538330078125, -21.350479125976562, -22.66985511779785, 1.8440767526626587, -14.014142036437988, -50.63905334472656, 10.572254180908203, 7.921850681304932, -1.0338255167007446, -12.72829818725586, 46.29335021972656, -20.268184661865234, 19.70294189453125, 38.27348709106445, -27.226594924926758, -23.343101501464844, 3.0695199966430664, 3.77567195892334, 22.7703800201416, -28.27149200439453, -23.212495803833008, -19.712718963623047, 28.61910057067871, -2.6693801879882812, -1.7941029071807861, -68.733154296875, -4.109109401702881, 12.661951065063477, -6.6401047706604, -18.310304641723633, 37.13597106933594, 13.17380142211914, 21.63294792175293, -48.76650619506836, 45.02314376831055, -40.911441802978516, 16.474040985107422, -14.990106582641602, -44.75909423828125, -25.000823974609375, 49.51676559448242, 35.24366760253906, -40.36895751953125, -5.714505195617676, 6.91273307800293, -6.109490394592285, 6.815208435058594, 45.54037094116211, -50.976810455322266, 7.663954734802246, -51.53247833251953, 29.875564575195312, -13.028308868408203, 0.12142197787761688, 14.196435928344727, -16.252803802490234, -19.620838165283203, 12.112200736999512, 3.2181644439697266, -49.84199523925781, -13.514229774475098, 6.658778667449951, 14.953593254089355, -28.912227630615234, 19.222759246826172, 7.925139904022217, -22.14171600341797, -56.45412063598633, 5.269830226898193, -3.5829014778137207, 7.047306060791016, 21.034526824951172, -4.6889328956604, 6.190970420837402, -29.783376693725586, 3.746668577194214, -35.76694869995117, -10.734649658203125, 16.99159812927246, -9.527759552001953, 50.22251892089844, -49.34331130981445, -19.072046279907227, -11.80789852142334, 20.491519927978516, 32.455543518066406, 16.405614852905273, -4.175732135772705, 32.98960876464844, -25.561397552490234, -34.09413528442383, -27.853469848632812, -2.409325361251831, 2.8938732147216797, 54.30261993408203, -30.453332901000977, 2.0869340896606445, -44.402809143066406, -17.22393226623535, 69.33877563476562, -24.76292610168457, -23.757261276245117, -7.585402488708496, -34.54832458496094, -42.17186737060547, -40.57929611206055, -24.79436683654785, 27.994277954101562, 35.32024002075195, 43.879737854003906, 73.72177124023438, -28.73741340637207, -19.995988845825195, -17.225666046142578, -50.08877182006836, -39.34501647949219, 19.271451950073242, -13.230713844299316, 9.529892921447754, 36.569026947021484, -15.911591529846191, -38.511680603027344, 7.565229415893555, -10.456491470336914, -5.738524913787842, -33.09568405151367, 11.685638427734375, -15.690499305725098, 14.742920875549316, 14.211039543151855, -18.4940185546875, 6.405819892883301, -21.697998046875, 19.956321716308594, 19.07379722595215, -29.654346466064453, -31.529552459716797, -27.0200138092041, 58.95413589477539, 11.93501091003418, -28.8503360748291, 16.758981704711914, -41.62678146362305, 29.563983917236328, 61.2072868347168, 3.7445292472839355, 5.419672012329102, 50.5784797668457, -15.910962104797363, 25.571258544921875, -0.5626168251037598, 8.200963973999023, 6.181812763214111, 5.4295783042907715, 68.05552673339844, 19.887800216674805, -40.40167999267578, -87.80810546875, 15.654289245605469, -36.02216339111328, 0.2228398472070694, 16.494367599487305, -35.896297454833984, 22.305217742919922, 24.493860244750977, -17.273639678955078, 28.0577449798584, -20.881803512573242, -2.8172192573547363, -21.0909366607666, 16.62517738342285, 47.101322174072266, 14.024211883544922, 12.634599685668945, -0.6864270567893982, -69.07213592529297, 42.480220794677734, 20.95783042907715, -29.362220764160156, -10.739013671875, -27.317861557006836, -57.734352111816406, 63.748077392578125, 28.211284637451172, -23.07945442199707, 29.29107093811035, 35.08903503417969, -3.9632365703582764, 5.248472690582275, -1.3906270265579224, -48.64720916748047, -9.655086517333984, 34.5159912109375, 21.274248123168945, -54.03036880493164, -4.915205955505371, -39.697235107421875, -31.01514434814453, -18.14196014404297, 9.508512496948242, -56.73346710205078, 33.80073928833008, -32.27934265136719, -10.403736114501953, 18.605388641357422, -16.27486801147461, -25.038188934326172, -12.92487621307373, -31.99397850036621, -13.935887336730957, -7.744635581970215, -30.716278076171875, -2.779256820678711, 14.783506393432617, -25.489486694335938, -0.0340505912899971, -45.54425811767578, 14.914687156677246, 12.353057861328125, -34.141014099121094, 5.070980548858643, -37.03535842895508, 4.329432487487793, -58.48420333862305, -9.242526054382324, 7.991424560546875, -1.0040154457092285, -31.855087280273438, -29.313695907592773, 11.615840911865234, -5.420337200164795, -13.79294490814209, 21.787647247314453, -38.2147216796875, -30.569515228271484, 28.232698440551758, -3.298649311065674, 8.250870704650879, 15.377898216247559, 0.4685916602611542, -4.76967716217041, 0.7932432889938354, 20.815710067749023, -14.607117652893066, 14.544760704040527, -5.2593793869018555, 27.80394172668457, 9.487906455993652, 15.236141204833984, -18.68000602722168, 27.632020950317383, -40.90915298461914, -32.256893157958984, 15.038697242736816, -1.0410609245300293, 17.187423706054688, 9.134620666503906, 15.672410011291504, -56.16640090942383, 1.2163078784942627, 1.148289442062378, 24.031253814697266, -28.408756256103516, -11.927475929260254, 18.876367568969727, 29.748207092285156, -66.57266235351562, -5.035096645355225, 14.556302070617676, 48.17438507080078, -27.29374885559082, 10.775915145874023, -49.87812042236328, 78.19136047363281, -15.041393280029297, -6.496891021728516, -22.48345184326172, -13.527925491333008, -2.026118278503418, 21.4739990234375, 42.044944763183594, 20.766313552856445, -13.56046199798584, 32.22240447998047, -10.759819030761719, 14.518376350402832, 44.39790344238281, 41.9339485168457, 54.934513092041016, -32.04154968261719, -15.250243186950684, -10.760000228881836, -43.83590316772461, -42.5082893371582, -17.52010726928711, 9.367480278015137, 51.50094223022461, -0.8627364635467529, -13.009997367858887, -12.653093338012695, 11.73863697052002, 51.71826934814453, -18.01575469970703, -3.8298654556274414, -6.407912731170654, 15.775126457214355, 1.7391831874847412, 7.778602600097656, 10.060100555419922, 1.5642330646514893, 9.010308265686035, -25.776260375976562, 20.7763671875, -5.41198205947876, 20.006933212280273, -47.58757019042969, -14.09416675567627, 48.27039337158203, 8.619568824768066, 13.303504943847656, -14.490772247314453, 54.61785125732422, -1.0981671810150146, 33.05506896972656, -22.908477783203125, 51.16753005981445, 7.311912536621094, 3.655913829803467, 28.437332153320312, -18.25766944885254, -53.72785568237305, -0.30302727222442627, 25.74825668334961, 21.020782470703125, 6.271843910217285, -8.150766372680664, -12.843905448913574, 44.166908264160156, -23.80037498474121, 11.959762573242188, 20.275758743286133, -37.41560363769531, 30.979825973510742, 26.77444076538086, -12.880093574523926, -3.161600112915039, 32.24176025390625, 25.302412033081055, -33.2799072265625, -2.167264938354492, -50.08802032470703, 30.520065307617188, -4.379098892211914, -29.267091751098633, 22.68229866027832, -19.516530990600586, 40.86524963378906, 44.05405044555664, 38.449485778808594, 32.11411666870117, -80.98097229003906, -23.902631759643555, -64.9568099975586, 20.714473724365234, -49.75567626953125, 6.006527900695801, -47.217247009277344, 9.574006080627441, -27.13462257385254, -50.24948501586914, -56.66358947753906, -14.937518119812012, 41.82448959350586, -5.454448699951172, 49.880958557128906, -12.426620483398438, -1.5045408010482788, 22.025524139404297, 7.400963306427002, -23.200822830200195, -16.948410034179688, 14.8682861328125, 15.384934425354004, 13.162272453308105, 31.245559692382812, -1.266098976135254, 2.661130428314209, 14.941521644592285, -9.487920761108398, -34.07038116455078, -9.127386093139648, -8.115418434143066, 34.80672073364258, -19.931257247924805, 20.01210594177246, 51.272125244140625, 11.342337608337402, -25.587533950805664, -18.33445930480957, -16.03786277770996, -21.31829071044922, -45.892791748046875, 3.5931594371795654, 21.65376091003418, -10.004024505615234, -4.584283351898193, 6.400148868560791, -15.972639083862305, -20.438566207885742, 33.82243347167969, 18.824718475341797, -31.497703552246094, -19.826997756958008, 30.144620895385742, -78.97518157958984, -67.73007202148438, 11.221208572387695, 61.27009582519531, 17.029813766479492, 59.34376525878906, 80.94583892822266, 21.646739959716797, -42.39193344116211, 12.259482383728027, -27.232711791992188, -0.45564010739326477, 3.204591989517212, -4.989218711853027, -59.048824310302734, -34.798362731933594, 33.8875846862793, 9.387833595275879, -3.7140207290649414, 15.09377384185791, 4.493396759033203, 33.708961486816406, 26.399585723876953, -17.119680404663086, -64.48934173583984, 61.28523635864258, 3.559135913848877, 18.887786865234375, 3.0191164016723633, -36.254554748535156, -11.621870040893555, 12.996819496154785, 4.023580551147461, -10.354913711547852, 61.46420669555664, 26.691726684570312, 18.718889236450195, 6.146791934967041, 17.184431076049805, 13.03359603881836, -31.730783462524414, 22.623483657836914, 2.0189123153686523, -20.37044334411621, 7.905313014984131, 53.28102493286133, -8.9017915725708, -56.525482177734375, -5.166491985321045, 39.841434478759766, -11.672198295593262, 27.440330505371094, -14.384049415588379, -3.349531412124634, 39.706581115722656, 32.45930480957031, 14.270286560058594, 3.385283946990967, -48.93808364868164, -10.871706008911133, -15.724827766418457, -5.6535444259643555, -2.626873731613159, -1.2652522325515747, -17.901405334472656, 7.255793571472168, 14.927828788757324, 108.42318725585938, 18.645618438720703, 16.97559928894043, -30.326610565185547, 59.13267135620117, 13.086267471313477, 55.34351348876953, 52.42478942871094, -33.41720962524414, -15.039142608642578, 22.797204971313477, -49.62424087524414, 2.2255001068115234, 36.70896911621094, -49.351707458496094, 14.540969848632812, -8.004536628723145, -30.619848251342773, -5.082098007202148, -27.728513717651367, 56.042545318603516, 35.05131149291992, 10.09265422821045, -22.689014434814453, -37.172428131103516, 25.82876968383789, 29.545486450195312, 11.240856170654297, 18.09463882446289, -17.30047607421875, 13.104605674743652, -24.537731170654297, -4.610645771026611, 34.28718948364258, -7.465544700622559, -16.834793090820312, 30.802932739257812, -30.046995162963867, 35.835201263427734, -33.298614501953125, 11.390153884887695, -2.008826494216919, -9.484827995300293, 23.42046356201172, -22.34966278076172, -47.815093994140625, -2.0280308723449707, -19.211368560791016, 29.88666343688965, -5.2831830978393555, -18.913677215576172, -31.154537200927734, 4.085949897766113, 9.330093383789062, 33.40742874145508, -17.936460494995117, -7.472170829772949, 4.437118053436279, 31.46012306213379, 66.37442779541016, -5.313728332519531, 27.68648338317871, 28.52219581604004, 29.089035034179688, -3.5798728466033936, -17.644746780395508, 14.174638748168945, 6.021712303161621, -5.4373979568481445, -10.683554649353027, -3.121922492980957, 13.673417091369629, -27.496936798095703, -25.682056427001953, 5.170509338378906, -11.606097221374512, -25.186687469482422, 5.991799831390381, 20.33551597595215, 5.399878025054932, 23.340604782104492, -22.78830337524414, -10.726601600646973, -12.215080261230469, 24.14990997314453, 17.233699798583984, 0.9330885410308838, 2.430732488632202, -0.388271301984787, -45.47748565673828, -35.32721710205078, -35.72755432128906, 5.929312705993652, -10.353987693786621, 4.64276647567749, -16.786962509155273, -42.734432220458984, -2.6734490394592285, -1.3415623903274536, -23.220659255981445, -7.6133198738098145, -1.938046932220459, -29.748085021972656, -57.16584014892578, -4.36447286605835, 15.231755256652832, -41.11678695678711, -38.729251861572266, -42.87045669555664, -23.10464859008789, -18.843435287475586, -47.238319396972656, -23.78192138671875, -18.581180572509766, -35.470603942871094, 39.63478469848633, 2.8466596603393555, 3.6750986576080322, -11.69270133972168, 18.699432373046875, -22.718997955322266, 22.474164962768555, 6.849833011627197, 10.692320823669434, -37.94051742553711, -9.146641731262207, 12.864339828491211, 80.11613464355469, 40.53623580932617, -23.55901527404785, 37.70966720581055, -28.891508102416992, -27.412792205810547, -52.70869445800781, 23.44812774658203, -4.067194938659668, -23.766559600830078, 40.96919250488281, -6.942804336547852, -58.28170394897461, 28.367219924926758, -48.019535064697266, 23.612268447875977, 27.313671112060547, -2.9383108615875244, -60.50979995727539, -7.1407389640808105, -39.95893859863281, -11.149587631225586, 0.12354877591133118, -42.48466110229492, 5.957985877990723, 23.49261474609375, 12.095914840698242, -11.310294151306152, -60.580936431884766, -5.478969573974609, 31.576828002929688, 52.612754821777344, 11.246284484863281, -37.95478820800781, 12.417962074279785, 9.006712913513184, -0.6075271368026733, 34.730464935302734, -36.22487258911133, 7.537998676300049, -19.003742218017578, -24.737817764282227, -4.1589035987854, 37.59449005126953, -27.16904067993164, 60.94928741455078, -42.667091369628906, -12.188318252563477, 1.980873465538025, 0.19280517101287842, -60.235084533691406, -0.915632963180542, 30.636159896850586, -6.804930210113525, -12.444769859313965, -48.76832962036133, 43.27128982543945, 30.238269805908203, 33.45320129394531, 29.638774871826172, -46.09413528442383, 21.301025390625, -44.951995849609375, 3.267143964767456, 0.4675772786140442, -12.8987398147583, 9.469430923461914, 12.518158912658691, -17.207420349121094, 21.768898010253906, 24.00543785095215, 34.56660079956055, -23.083580017089844, 34.654537200927734, -36.06200408935547, -33.3143310546875, -39.0782356262207, -45.48465347290039, 25.25295066833496, -16.59082794189453, 10.104619026184082, -41.163551330566406, -22.876514434814453, 16.94017791748047, -36.62122344970703, -22.683378219604492, 31.767662048339844, -24.676454544067383, -54.71125793457031, -74.37268829345703, -23.692522048950195, -7.811001300811768, -33.41413116455078, -4.343989849090576, -33.081336975097656, 9.576181411743164, 14.603109359741211, -53.98809814453125, 30.574987411499023, -73.96652221679688, 53.113548278808594, -15.629570007324219, -58.603904724121094, 18.99528694152832, -30.32950782775879, 5.474997043609619, 96.96363830566406, 3.044677257537842, -36.80329132080078, -19.961410522460938, -4.279979705810547, 28.82235336303711, 45.503929138183594, -21.143497467041016, -18.54818344116211, 5.731074333190918, 52.38163757324219, 39.66700744628906, 46.66033172607422, 12.748943328857422, 11.323287963867188, 42.85792541503906, -26.558517456054688, -52.38629150390625, 5.606846332550049, -7.870582103729248, -1.1975582838058472, -17.58242416381836, -9.21411418914795, -5.404666423797607, -43.76081848144531, -42.871944427490234, -18.823917388916016, -59.91374206542969, -27.596952438354492, -10.94571590423584, -13.53168773651123, -22.86338233947754, 20.97435760498047, 32.38935852050781, 11.002481460571289, -12.552072525024414, -7.004384994506836, -59.89110565185547, -12.346232414245605, 47.89120101928711, 16.920473098754883, -52.72738265991211, 36.25776290893555, 11.768853187561035, -20.992483139038086, -48.81987762451172, -16.296466827392578, -16.464879989624023, 11.642416954040527, 13.536422729492188, 13.571807861328125, 49.38225555419922, 6.772782325744629, 0.20076817274093628, -17.925966262817383, 34.734107971191406, 34.4485969543457, -24.722654342651367, 17.313751220703125, 28.391666412353516, 3.10697340965271, -60.56496047973633, -3.338270902633667, -46.5196647644043, 28.776716232299805, 7.932921886444092, 37.99211883544922, 14.514419555664062, -49.47471618652344, -4.454466342926025, 46.67949295043945, 40.399959564208984, -13.942842483520508, -7.0102972984313965, 25.165489196777344, -22.8889102935791, -26.20461082458496, -59.115943908691406, 1.3049366474151611, -1.6210614442825317, 43.46217727661133, 8.33756160736084, -7.841538429260254, -16.568880081176758, -18.418960571289062, 11.545836448669434, -53.623985290527344, -51.858192443847656, 39.47910690307617, -98.61730194091797, 11.029413223266602, 36.58354949951172, -56.63646697998047, -48.23979187011719, 56.132484436035156, 2.0708513259887695, 9.959168434143066, 9.509063720703125, 13.516833305358887, -4.066899299621582, 56.50175094604492, -34.11235046386719, 6.360472202301025, 49.99614715576172, -35.4622917175293, -28.653614044189453, -21.800508499145508, 6.690131664276123, 19.748424530029297, -28.902870178222656, 7.669290542602539, 52.967979431152344, 28.73202896118164, -16.737581253051758, 5.915090084075928, 15.809179306030273, -37.398681640625, -5.633658409118652, 28.587074279785156, 23.27924919128418, 22.781661987304688, -15.118861198425293, -0.2353149950504303, -59.9136962890625, 34.38798904418945, -19.152381896972656, 1.2400182485580444, 9.588772773742676, -70.06182861328125, 27.200408935546875, -10.974111557006836, 20.993383407592773, -46.369712829589844, 26.954082489013672, 25.98001480102539, 5.995176792144775, 12.341469764709473, -6.8696675300598145, -5.460630416870117, -10.624561309814453, 32.298831939697266, -30.439964294433594, -45.78749084472656, -25.635446548461914, 7.037900447845459, -0.7106162309646606, -0.9388610124588013, 81.70235443115234, 22.245100021362305, -45.39274978637695, 31.613109588623047, 53.442291259765625, -11.438952445983887, 52.76807403564453, 50.503662109375, 39.98368835449219, -26.159406661987305, 17.340251922607422, -33.316131591796875, -40.74416732788086, -31.504150390625, -14.207748413085938, 3.030585765838623, 41.41168212890625, 29.800621032714844, 3.703130006790161, -22.998464584350586, -18.271467208862305, -16.46744155883789, 20.864437103271484, -34.36996841430664, -44.22572326660156, 48.067787170410156, 18.540367126464844, -6.977419376373291, -21.890958786010742, 45.166141510009766, -10.735212326049805, -7.606924533843994, 28.133203506469727, 12.273825645446777, 29.61570930480957, -55.26990509033203, 26.243059158325195, 26.527870178222656, -20.031875610351562, 49.82512664794922, -7.743514060974121, 10.282108306884766, -20.55836296081543, -28.994768142700195, -8.6798734664917, -40.30064392089844, -22.741941452026367, -59.00843048095703, -35.5726432800293, -4.3213934898376465, 3.4519875049591064, -18.573640823364258, -9.228275299072266, 6.469380855560303, -24.564054489135742, -60.57582473754883, 20.00924301147461, -6.863073825836182, 4.120711803436279, -66.69839477539062, 1.9732531309127808, 32.745487213134766, 28.480226516723633, -1.3760693073272705, -33.379493713378906, -51.78318786621094, -11.215158462524414, -16.233257293701172, 31.518413543701172, 16.387767791748047, 0.571438193321228, 18.25737190246582, 15.265825271606445, 13.866695404052734, 3.1446521282196045, 16.243785858154297, 7.405050277709961, 16.152990341186523, 18.72376823425293, -32.45269775390625, -2.7810587882995605, 34.50647735595703, -15.315521240234375, 8.191424369812012, 28.828725814819336, -13.006844520568848, -6.135529518127441, 14.016040802001953, -51.67774200439453, -17.55689811706543, -17.016504287719727, 32.28593063354492, 19.72464370727539, 32.141937255859375, -6.951070785522461, 12.234384536743164, 9.2033052444458, -12.55488395690918, 19.953155517578125, 3.620635509490967, -21.02678680419922, 18.927831649780273, -14.401835441589355, 63.68531799316406, -12.802647590637207, 60.69126510620117, 11.9330415725708, -12.432369232177734, -34.14915466308594, -38.41117858886719, 9.190369606018066, 76.24506378173828, -17.513029098510742, -14.888904571533203, -54.51097106933594, -33.86198806762695, -13.43681526184082, -8.053383827209473, -14.25265884399414, -26.304487228393555, -8.248223304748535, 28.00859832763672, 0.9538487195968628, -68.22087097167969, -11.744729995727539, -16.371337890625, 3.3590075969696045, 34.24271011352539, 21.49404525756836, -61.18699645996094, -18.394750595092773, 5.042232513427734, 0.3763173520565033, 28.04339027404785, -4.750504970550537, -20.80299949645996, -18.198081970214844, -25.610090255737305, 15.598660469055176, 62.46617126464844, 3.4647202491760254, 5.012686729431152, -36.418212890625, 11.824943542480469, 0.521354079246521, 9.459498405456543, -8.527535438537598, 31.487567901611328, 46.710716247558594, 15.330199241638184, -15.974637985229492, -45.20206832885742, -51.62476348876953, 4.237008094787598, 41.753929138183594, -7.683715343475342, -36.58195495605469, 19.922813415527344, -16.201631546020508, 23.885351181030273, 31.189098358154297, 25.040138244628906, 44.95928192138672, -7.915974140167236, -15.528326988220215, -15.726837158203125, -41.9454345703125, -25.380582809448242, -5.623098373413086, -6.913634777069092, -35.11688232421875, 28.520662307739258, -40.00617218017578, -12.080169677734375, -47.50725555419922, -3.811554431915283, -37.042724609375, -40.29902648925781, 44.02650451660156, 20.54015350341797, -11.073310852050781, -15.847064971923828, 25.479806900024414, -11.334980010986328, -44.45358657836914, 32.00878143310547, 34.584075927734375, -3.5761234760284424, 8.769495010375977, 33.6233024597168, 7.747566223144531, 13.017619132995605, 35.183006286621094, 10.531327247619629, 39.00261306762695, -20.469539642333984, -12.750238418579102, -47.68367004394531, 23.6662654876709, 30.321697235107422, -17.8955078125, -2.506434440612793, -2.143477439880371, 34.72758483886719, -33.5172119140625, -18.890623092651367, -44.88541030883789, -32.164520263671875, -17.01319694519043, 8.941502571105957, -36.68173599243164, -7.846380233764648, 24.25855827331543, -40.03190231323242, 6.4190263748168945, -19.337968826293945, 23.259876251220703, -15.362225532531738, 64.09632110595703, -10.587891578674316, -21.264291763305664, 38.745277404785156, -8.56383228302002, 25.851757049560547, 5.296650409698486, -13.219792366027832, -13.049388885498047, 22.418392181396484, 56.76012420654297, -13.220726013183594, 17.892208099365234, 26.662927627563477, -12.88502311706543, 44.783203125, 1.876255989074707, 9.047545433044434, 7.271485805511475, -4.651756763458252, 10.252306938171387, -22.628437042236328, 38.867393493652344, -7.481833457946777, -5.396712779998779, -28.048519134521484, 54.82080841064453, 33.40752410888672, -13.843987464904785, -35.40758514404297, 3.806985378265381, 14.21955680847168, 28.951461791992188, -22.54199981689453, -22.609079360961914, 0.19395214319229126, 37.37104034423828, 55.42787170410156, -11.754324913024902, 3.3856005668640137, -23.12996482849121, 1.709507703781128, -3.7263755798339844, 17.41228485107422, -0.44691142439842224, -27.699975967407227, -18.193769454956055, 29.362638473510742, 19.896451950073242, 54.2805290222168, 13.43381404876709, 5.469358921051025, 28.524616241455078, 39.352359771728516, 13.999187469482422, 8.189468383789062, 29.1904354095459, -50.4759521484375, 26.87677574157715, -15.448674201965332, 58.29033660888672, -32.02116394042969, -4.2446722984313965, -20.571317672729492, -45.73517608642578, -7.808742046356201, -13.471067428588867, -50.88922882080078, 0.8098119497299194, -70.13180541992188, 19.17803192138672, -2.5701487064361572, 5.385097503662109, -30.734466552734375, 8.382234573364258, -34.804893493652344, -18.302841186523438, -11.562261581420898, 21.994861602783203, -38.75149917602539, -52.69743347167969, 19.02356719970703, -16.369306564331055, -10.761277198791504, 26.02391242980957, 0.326322466135025, -19.006433486938477, 7.981161594390869, -56.115814208984375, 12.997734069824219, -32.83515548706055, -14.635641098022461, 13.438469886779785, -10.745634078979492, -39.332035064697266, -15.735438346862793, 25.576364517211914, -1.082849144935608, 13.173274993896484, -26.156105041503906, -26.192461013793945, 1.6177152395248413, 10.691071510314941, 9.840448379516602, 0.11901162564754486, -17.137073516845703, 4.489163398742676, 49.1104736328125, -4.884012222290039, 2.1533093452453613, 11.176366806030273, 45.000667572021484, 25.31679344177246, 35.82597732543945, -16.23295783996582, 28.304462432861328, -18.907503128051758, 33.70502853393555, -9.575862884521484, -27.30054473876953, 7.414994716644287, 44.918514251708984, 22.472490310668945, 17.49483299255371, -13.047257423400879, -22.398454666137695, 23.92233657836914, -12.881165504455566, -25.15123748779297, 40.96002960205078, 41.47465896606445, -4.988221168518066, 12.52074146270752, -4.406427383422852, 16.671432495117188, 4.6713337898254395, 59.931644439697266, 3.6701669692993164, 34.18144989013672, -54.21302795410156, 5.615747928619385, 49.722007751464844, -17.060863494873047, 11.949823379516602, -25.440513610839844, -37.85484313964844, 17.223291397094727, -24.777742385864258, 20.56104850769043, -11.63078784942627, 17.21700668334961, 34.5108757019043, -15.761465072631836, -53.784339904785156, 1.156688928604126, 0.6423589587211609, -10.196093559265137, -37.04550552368164, 2.8153932094573975, 58.34794998168945, -34.807289123535156, 33.25798797607422, 24.19742774963379, 8.391515731811523, 35.146095275878906, 1.8697963953018188, 38.1452751159668, 35.01374435424805, -16.00262451171875, 30.7510929107666, -27.770160675048828, -94.83728790283203, 17.25664710998535, 38.863006591796875, -12.39918041229248, -21.223358154296875, 9.180872917175293, 32.11601638793945, 1.2102736234664917, -17.41709327697754, -9.884815216064453, -32.860145568847656, -8.587491035461426, 22.472976684570312, 18.190753936767578, 9.57680892944336, -5.465409755706787, 23.191688537597656, -25.503751754760742, 20.970844268798828, 16.824880599975586, -15.34665584564209, -26.723909378051758, 3.6705431938171387, -48.65012741088867, -12.801583290100098, -69.79763793945312, -32.00931930541992, -38.11566162109375, -0.6566847562789917, 50.59097671508789, 3.95329213142395, 10.774810791015625, 27.17074966430664, 19.755022048950195, 26.598947525024414, -46.98389434814453, 61.933685302734375, -32.19695281982422, 52.48565673828125, -18.090003967285156, 9.278151512145996, 20.232337951660156, -8.577051162719727, -9.619193077087402, 26.996673583984375, 70.4975814819336, -5.982783317565918, -21.52152442932129, 29.841737747192383, 22.838741302490234, -18.782150268554688, 40.53052520751953, 18.488975524902344, 6.406068801879883, 13.957201957702637, -19.027095794677734, 51.2866325378418, 3.138519525527954, 17.072540283203125, -1.839040756225586, 25.236495971679688, 6.835904598236084, -17.962024688720703, 1.9409412145614624, -35.05023956298828, 17.347993850708008, 17.210371017456055, 2.220845937728882, -30.61610984802246, -32.303924560546875, -21.162410736083984, 14.904687881469727, 23.12278175354004, 10.279026985168457, -7.7663068771362305, -21.01701545715332, -24.801284790039062, 24.197769165039062, -0.9661228656768799, 20.526023864746094, -16.051738739013672, 23.289127349853516, -35.54367446899414, -24.290584564208984, -0.989675760269165, 10.9982271194458, -46.419700622558594, 14.20796012878418, -83.79277801513672, -67.20658111572266, 1.3527926206588745, 8.613109588623047, -6.970854759216309, -30.880901336669922, 23.69562339782715, 12.429617881774902, 68.48986053466797, -34.2049446105957, -0.8859238028526306, -11.158644676208496, -14.72864055633545, -41.42809295654297, 44.219627380371094, 4.935227394104004, -59.46434783935547, 37.479713439941406, -7.144830703735352, 16.290735244750977, -55.393245697021484, 0.24587729573249817, -25.34239387512207, 33.60136795043945, 18.81748390197754, -7.035797595977783, 9.595396041870117, 15.522794723510742, -19.350296020507812, -5.354550838470459, 41.806602478027344, -21.676979064941406, -19.373268127441406, 47.1445426940918, -5.572414875030518, -2.7711682319641113, 22.35191535949707, -16.379531860351562, 3.4716360569000244, -27.13896942138672, -19.686758041381836, -23.703628540039062, 8.343451499938965, -26.49529266357422, -31.546415328979492, -34.069576263427734, 47.823123931884766, -20.804664611816406, -13.96735954284668, 14.59665584564209, -11.202795028686523, 38.13773727416992, -5.367785453796387, -0.39357638359069824, -14.805100440979004, 19.285621643066406, -20.62921905517578, -17.411632537841797, -17.25745391845703, -5.031767845153809, -33.14152145385742, -13.779190063476562, 1.8957784175872803, -34.04731750488281, 1.2656641006469727, -26.41359519958496, 18.0471248626709, 36.564998626708984, 48.1109504699707, 2.2156729698181152, 5.9326934814453125, -32.89908981323242, 13.149456024169922, -9.249490737915039, 22.744291305541992, -20.176544189453125, 2.574920654296875, -14.731672286987305, -4.905675411224365, 32.53351593017578, -44.93592071533203, 13.35926342010498, 11.906596183776855, -2.5674421787261963, -2.618666410446167, -41.52280807495117, -10.073932647705078, 3.9689137935638428, 25.824453353881836, -0.8707259893417358, -48.96282196044922, -7.619075775146484, 33.659080505371094, -53.55242156982422, 20.37639808654785, -31.223970413208008, -9.434710502624512, -7.7120795249938965, 6.8068928718566895, 1.0080701112747192, 45.88009262084961, 47.035430908203125, -38.22843551635742, 4.2057719230651855, 3.136636734008789, 6.833284854888916, 26.29510498046875, -23.926502227783203, 26.059232711791992, -4.052477836608887, -3.317702531814575, 44.241371154785156, -16.926485061645508, 23.292081832885742, -40.891014099121094, -28.938474655151367, 14.269574165344238, 51.315101623535156, 11.651594161987305, 6.132148742675781, -30.472972869873047, 12.864189147949219, -1.1222493648529053, 13.185720443725586, 6.087513446807861, 13.979323387145996, 2.3317501544952393, -31.728307723999023, -10.295655250549316, -18.12007713317871, -31.357555389404297, 30.043025970458984, -56.82988739013672, -18.684581756591797, -7.711905479431152, -18.183738708496094, 7.884416103363037, -0.92173832654953, -27.82550621032715, 23.82310676574707, 3.551182270050049, -24.801136016845703, 6.688368320465088, -6.034977436065674, -8.328109741210938, 38.73247528076172, -5.749668598175049, -57.464599609375, 20.409292221069336, -15.641079902648926, 17.82258415222168, 18.70513153076172, 6.032268524169922, 69.5140380859375, -2.2067315578460693, -4.605103969573975, -30.205957412719727, -76.95509338378906, -13.228635787963867, 33.78153610229492, -21.305862426757812, 23.57476806640625, 30.141265869140625, -16.03644561767578, 26.934804916381836, -5.514449119567871, -42.18032455444336, -61.64069747924805, 41.40365219116211, 31.229724884033203, -77.18484497070312, -31.160585403442383, 53.363487243652344, -30.773834228515625, 4.088659286499023, -17.97709083557129, -7.180935382843018, 19.462112426757812, 18.244600296020508, -49.7677116394043, 48.18404006958008, -5.740609169006348, -36.33718490600586, -2.063117504119873, 13.814208984375, 29.999849319458008, -11.505951881408691, 3.879873037338257, -9.647680282592773, -31.380525588989258, 81.08734130859375, -0.776836633682251, 14.731390953063965, -47.24065017700195, 22.446197509765625, 33.09750747680664, -6.542390823364258, -72.32975006103516, 53.687252044677734, 1.3740875720977783, -39.67041778564453, 17.4908504486084, -34.68279266357422, 25.827756881713867, -45.80160903930664, 17.283388137817383, -55.62737274169922, -4.931154727935791, -22.009607315063477, -23.75292205810547, -56.2139892578125, 14.647378921508789, 39.71882629394531, -1.5250600576400757, -24.81435775756836, -47.266231536865234, -14.89760971069336, -4.236902236938477, -28.487152099609375, 27.9537353515625, 6.5389509201049805, -18.657381057739258, -3.569131374359131, 52.15434646606445, 10.847870826721191, 38.103614807128906, 32.47526931762695, -15.568753242492676, 56.3241081237793, -25.10807228088379, 37.87436294555664, 40.145233154296875, -6.596780776977539, 6.121793270111084, -30.866289138793945, 24.674882888793945, 37.04854202270508, 9.917819023132324, -29.463542938232422, -6.847861289978027, 21.677324295043945, 9.387173652648926, 30.910737991333008, -20.941953659057617, 6.567443370819092, 20.387420654296875, -27.518712997436523, 4.902973175048828, 6.0714497566223145, 9.011220932006836, -19.2393798828125, 12.455865859985352, -34.449344635009766, -3.245997428894043, 52.12059783935547, -54.73857879638672, 33.52284240722656, 38.546424865722656, -2.1172969341278076, -81.13460540771484, -42.74576950073242, 5.539926528930664, -45.78749084472656, -3.602510452270508, -8.170496940612793, -54.25899124145508, -14.424254417419434, 4.639545917510986, 27.44866371154785, 57.799583435058594, -35.612159729003906, -6.375135898590088, -6.329321384429932, 5.057519912719727, 39.75485610961914, 44.191802978515625, -27.68132781982422, 8.662601470947266, -49.94390869140625, 16.342134475708008, -13.641655921936035, -52.667232513427734, -4.489711761474609, -2.5437991619110107, -28.322574615478516, 20.616077423095703, 12.975567817687988, 26.492582321166992, -19.856138229370117, 24.603511810302734, 24.624980926513672, 29.199979782104492, -20.884653091430664, 37.077392578125, 30.48370933532715, 11.38827896118164, -23.730981826782227, 24.915536880493164, -20.451313018798828, 35.39936447143555, 48.369285583496094, 67.83456420898438, 15.092135429382324, -5.8149309158325195, 49.439476013183594, 31.83635139465332, -89.72161102294922, -29.914873123168945, 11.387791633605957, 5.723793029785156, -33.01722717285156, 14.432038307189941, 1.5738791227340698, -40.32001876831055, -67.37503814697266, -68.82056427001953, 7.784918785095215, -0.10909511148929596, 18.723466873168945, 5.397435188293457, 13.474614143371582, -47.665897369384766, -13.137438774108887, 14.408087730407715, -16.1302433013916, -21.808164596557617, -16.919849395751953, 27.53323745727539, 2.5650949478149414, -50.72763442993164, -4.7809929847717285, -58.351383209228516, 3.524214506149292, 3.4512064456939697, 11.585482597351074, -9.697212219238281, 45.65408706665039, 33.02572250366211, -46.482242584228516, 2.550621271133423, -10.346351623535156, -6.298524379730225, -21.36701011657715, 49.28669357299805, 22.609527587890625, -3.6038784980773926, 30.12128257751465, 48.350746154785156, -18.623306274414062, 71.97103118896484, -37.7332878112793, -42.18553924560547, 53.32598876953125, 78.56814575195312, 83.57762145996094, 16.981201171875, 9.088068962097168, 19.5145263671875, 6.5460100173950195, -0.5546088218688965, 23.654497146606445, -5.931354522705078, -28.0897159576416, -0.45354828238487244, -16.219669342041016, -33.045860290527344, 40.33132553100586, -11.173700332641602, 11.723857879638672, 45.06346893310547, -23.103849411010742, -5.32094669342041, -3.7405810356140137, 4.675768852233887, -40.27888488769531, 39.357086181640625, 15.834485054016113, 8.603377342224121, -3.601425886154175, 53.69475173950195, 30.25630760192871, 1.1184916496276855, 38.97422790527344, -36.06648254394531, -21.985002517700195, 0.07024809718132019, -10.288008689880371, 17.44070816040039, 3.405402660369873, -18.81074333190918, 15.0227632522583, -9.032957077026367, -22.786006927490234, 16.65874671936035, -71.94489288330078, 0.4629340171813965, 40.93136215209961, 34.195682525634766, 18.175975799560547, -11.151187896728516, -4.441535949707031, -7.255867958068848, 9.588353157043457, 6.101715564727783, -26.997100830078125, -29.029067993164062, -61.17107391357422, 75.67262268066406, 14.835770606994629, -10.590060234069824, -41.59251022338867, 14.103035926818848, -0.4007379412651062, -11.022560119628906, -75.9287109375, 7.587203025817871, 22.962596893310547, -2.986396551132202, -28.12843132019043, -7.151008605957031, -33.529197692871094, -23.93086051940918, -5.726291179656982, -9.574175834655762, 12.842669486999512, -34.26484680175781, 48.682212829589844, -29.119985580444336, 35.621700286865234, 32.38071060180664, -44.29582214355469, -32.7877197265625, 25.941686630249023, 6.123688697814941, -3.027190685272217, -49.95421600341797, 34.11791229248047, -8.541077613830566, 17.774198532104492, 29.639495849609375, 12.340270042419434, -4.772168159484863, -11.00429630279541, -38.235511779785156, 10.69582462310791, -12.034415245056152, -29.922550201416016, 23.160181045532227, -12.586938858032227, 0.017627974972128868, 78.37606811523438, 22.33571434020996, -51.75212478637695, 7.860546112060547, 24.5174503326416, 22.495710372924805, -7.305028438568115, -4.951790809631348, -10.433553695678711, -22.675540924072266, 22.360246658325195, -12.083830833435059, 49.52015686035156, 49.04732894897461, 56.273738861083984, 2.7327046394348145, -12.95235824584961, -38.5389289855957, -27.397157669067383, -22.83390998840332, -13.181730270385742, 16.757198333740234, -11.061774253845215, 8.904012680053711, 10.575289726257324, -35.28904342651367, 15.873498916625977, -13.563272476196289, -17.117042541503906, -14.82107925415039, 12.208453178405762, -39.70652770996094, -35.73068618774414, -9.684715270996094, -9.091793060302734, -31.872089385986328, 0.6596048474311829, -23.372066497802734, -33.70177459716797, -39.274375915527344, 31.422550201416016, -17.568178176879883, -43.24388885498047, -44.84492874145508, 45.01667785644531, -0.80391925573349, 2.5052568912506104, 16.02116584777832, -23.844661712646484, 34.18143081665039, 10.248187065124512, -14.767183303833008, 13.301016807556152, -17.203723907470703, 41.208091735839844, -32.07927322387695, 34.16228485107422, -8.916196823120117, 10.221752166748047, 34.81011199951172, -1.260233998298645, 30.342178344726562, 47.94150924682617, -16.88993263244629, 39.91685104370117, 6.409766674041748, -28.53268051147461, 10.897357940673828, 4.545437812805176, 26.4910888671875, -23.030223846435547, 16.61056137084961, 37.73760986328125, 18.310443878173828, -46.869422912597656, -72.4670639038086, 23.189491271972656, 15.956315040588379, 42.813568115234375, -33.784523010253906, 7.847538948059082, 3.7576000690460205, 7.9408488273620605, -26.044368743896484, 20.197242736816406, 8.160685539245605, -10.026511192321777, 22.260568618774414, -8.444059371948242, 6.727006912231445, -82.33031463623047, -42.54788589477539, -57.1330451965332, 18.190649032592773, -0.6929036974906921, -20.61115074157715, 5.712925434112549, 8.286334991455078, -23.666120529174805, 22.108917236328125, -8.170677185058594, 12.789255142211914, 5.678004741668701, -54.48017501831055, -14.815327644348145, 49.527732849121094, 18.25118637084961, 11.652050971984863, 31.65102195739746, 33.045738220214844, -21.688629150390625, 77.27857971191406, -19.227313995361328, 12.192155838012695, 12.821897506713867, 17.435924530029297, -46.23902893066406, -24.53489112854004, -5.393886566162109, -7.459192276000977, -57.206905364990234, 74.87973022460938, 22.499021530151367, -4.2863383293151855, -36.10285568237305, -32.6740837097168, 4.4464263916015625, -60.792354583740234, -1.5517356395721436, 34.773685455322266, 0.8300084471702576, -21.44647216796875, 23.336511611938477, 6.387640476226807, 3.832296848297119, 27.3437442779541, -31.037582397460938, -40.695533752441406, 49.32388687133789, -29.994504928588867, 38.807647705078125, 7.102444171905518, 25.8439884185791, -23.510034561157227, 20.159027099609375, -24.618154525756836, -14.141491889953613, -50.64917755126953, 2.5374889373779297, -1.7467674016952515, -10.99875259399414, -66.92730712890625, 3.1927263736724854, -52.79152297973633, 40.98772430419922, 6.136640548706055, 2.7148749828338623, -7.805527687072754, 42.36101150512695, 3.4927377700805664, 19.51809310913086, -36.27568817138672, -2.7919039726257324, 49.83149337768555, -1.8785139322280884, -17.93642234802246, -15.28169059753418, 17.749340057373047, -10.177197456359863, -13.83940315246582, 1.1339330673217773, 8.34587574005127, -37.4791259765625, 6.586315155029297, 20.38258934020996, -42.74089813232422, 50.59663009643555, 26.70133399963379, -3.2853891849517822, -6.115397930145264, -51.643131256103516, -10.049492835998535, -81.85969543457031, -33.11846923828125, -70.68550872802734, 21.900632858276367, -29.674007415771484, 17.510643005371094, -40.79848098754883, -26.500534057617188, 27.113245010375977, 22.598173141479492, 14.895252227783203, 7.927418231964111, 10.051756858825684, 51.109798431396484, 2.4280173778533936, -30.80510902404785, 7.407644271850586, 1.7021995782852173, -11.906842231750488, 16.10544776916504, -40.705299377441406, -12.451189041137695, -5.9318318367004395, 27.261152267456055, 17.652584075927734, 14.956446647644043, -58.20508575439453, 41.099029541015625, -61.15097427368164, 28.610885620117188, -23.10434913635254, -41.742916107177734, 25.148164749145508, -8.962236404418945, -39.89122009277344, -13.440351486206055, -10.569440841674805, -1.3296164274215698, -23.30636978149414, 30.92416000366211, 77.76323699951172, -7.430973529815674, 45.751220703125, -26.150575637817383, -11.308236122131348, 9.601799011230469, 12.82550048828125, 21.781681060791016, -9.220240592956543, 12.539982795715332, -16.177560806274414, 19.3438663482666, 1.8403537273406982, 13.210733413696289, -20.33859634399414, -7.38716983795166, 9.675431251525879, 5.7028326988220215, 18.85063934326172, -5.782009124755859, 7.726442813873291, -9.297311782836914, 29.154142379760742, 16.306188583374023, 22.822431564331055, -25.22710609436035, 20.144306182861328, -21.697338104248047, -37.43439483642578, 11.589754104614258, 11.700132369995117, -2.308351516723633, 5.740439414978027, -4.023102760314941, -14.201236724853516, -30.306293487548828, 18.8917236328125, -19.202571868896484, -3.1807124614715576, 22.44417953491211, 59.15071105957031, 56.06662368774414, -7.884463310241699, -51.768348693847656, 26.95745849609375, 21.818119049072266, -1.144283413887024, 6.248096942901611, 11.591504096984863, -33.199344635009766, -24.042701721191406, 67.29327392578125, 5.096858978271484, 28.827655792236328, 0.023973021656274796, -19.99039077758789, 2.831940174102783, 17.292144775390625, -9.870954513549805, 24.076997756958008, 11.73118782043457, -46.63795471191406, 15.316993713378906, 5.261404514312744, -21.059524536132812, -10.92917251586914, -1.7744909524917603, 50.63613510131836, 42.04957580566406, -37.25798416137695, 18.081241607666016, -32.69446563720703, -30.25897979736328, -0.6955487728118896, -26.355531692504883, 31.813676834106445, 38.61343002319336, 11.81657600402832, 4.535712242126465, 25.895809173583984, -3.084270477294922, -5.904220104217529, 24.189485549926758, -41.26481246948242, 9.168305397033691, 4.838150978088379, -18.105426788330078, -2.2972490787506104, -9.880590438842773, 1.8917772769927979, 27.967269897460938, -28.14467430114746, -20.71240234375, -18.85344886779785, -1.3467937707901, 40.015499114990234, 63.78675079345703, 7.226475715637207, -11.803262710571289, 14.800281524658203, 23.74164581298828, -34.23721694946289, -11.797677993774414, 1.1830788850784302, 29.173738479614258, -26.80886459350586, 43.02413558959961, -37.14765930175781, -48.672542572021484, 11.75174617767334, 17.498275756835938, 25.874326705932617, 21.461891174316406, -12.212700843811035, 9.7838134765625, 0.4961940348148346, -29.50136947631836, 60.774627685546875, -15.709815979003906, 22.675718307495117, 14.36161994934082, -29.6324405670166, 11.073742866516113, -2.2106199264526367, 18.738855361938477, -7.430914402008057, 2.9335083961486816, -17.195049285888672, -15.902273178100586, -47.1802978515625, -6.564846992492676, -24.831981658935547, 46.21355056762695, 24.908124923706055, 44.15565872192383, 48.36062240600586, 23.21898651123047, -44.447532653808594, 15.486152648925781, 7.934343338012695, 54.035675048828125, -43.558685302734375, -45.79011917114258, 7.179910659790039, 28.54143524169922, -31.370670318603516, -26.866533279418945, 18.32727813720703, -47.83131408691406, 8.350118637084961, 14.358254432678223, -27.866512298583984, 56.881195068359375, 3.50030517578125, -1.2973220348358154, 9.50605583190918, -14.01952075958252, -26.716915130615234, -32.186119079589844, -34.27967834472656, -40.793190002441406, -37.078861236572266, 5.647280693054199, -17.759933471679688, 24.955102920532227, 30.262760162353516, 9.087841033935547, -10.360037803649902, -0.3571315407752991, -6.991039276123047, -1.0991809368133545, 11.069706916809082, 12.648606300354004, 20.75383758544922, -16.43619155883789, 3.973909378051758, 8.007505416870117, 0.9100767970085144, -48.114707946777344, 10.714338302612305, -0.4551289677619934, -49.9506721496582, -31.59624671936035, 1.8651127815246582, 33.706031799316406, 46.53642654418945, -83.70500946044922, -30.96875762939453, -19.027923583984375, -5.190866470336914, -19.77559471130371, 44.56340026855469, -3.3375244140625, -29.922183990478516, 60.958255767822266, -6.17940616607666, 39.558956146240234, -12.924552917480469, -12.094722747802734, 24.91396141052246, -72.3215103149414, 7.399044990539551, -54.46282958984375, 9.557517051696777, -9.396361351013184, -15.629112243652344, -19.690465927124023, -8.969697952270508, -19.62166976928711, 2.1700809001922607, -44.1422233581543, 24.07612419128418, -46.523353576660156, -1.066056728363037, -0.6185063123703003, -0.09090810269117355, 36.33927536010742, 16.146413803100586, 55.16349792480469, -23.607702255249023, 20.0118408203125, 18.850954055786133, 42.964656829833984, -27.947341918945312, 31.036998748779297, 9.675944328308105, -3.0727574825286865, 16.48410415649414, 27.56956672668457, -43.917083740234375, 22.93851661682129, 28.30809783935547, 27.80755043029785, 31.604503631591797, 61.335548400878906, 15.467771530151367, 65.69039916992188, -6.267704486846924, 22.96704864501953, 16.695615768432617, -3.3433899879455566, 25.685447692871094, -37.46221160888672, -24.874319076538086, -5.642109394073486, 22.069852828979492, 17.451465606689453, -36.07222366333008, 41.66378402709961, -35.39347839355469, -4.4994120597839355, 29.641109466552734, 20.83376121520996, 42.154685974121094, 23.325252532958984, 14.038819313049316, 38.107994079589844, 12.142844200134277, 40.482933044433594, 38.15154266357422, -6.534322738647461, -24.734373092651367, -12.56637954711914, -8.875468254089355, 9.82573127746582, 1.2094002962112427, -13.705841064453125, -62.102508544921875, -1.145878553390503, 19.308303833007812, 10.116573333740234, -27.07732582092285, -10.985145568847656, -7.151057720184326, -27.345853805541992, -8.640557289123535, 11.565637588500977, 9.21642780303955, 15.541250228881836, 24.665348052978516, -12.518987655639648, 26.60137176513672, -5.022996425628662, -0.5333055853843689, -17.44443702697754, 49.63594055175781, 20.39177131652832, -9.31248664855957, 25.888988494873047, 50.24956130981445, -36.833839416503906, 65.9404067993164, -24.329193115234375, 24.693819046020508, -59.17413330078125, -39.28679656982422, 68.25263977050781, -27.81884002685547, 7.258045673370361, 9.3936185836792, 87.03414916992188, 7.388376235961914, 26.31719398498535, -18.337501525878906, 36.136940002441406, 16.82516098022461, 14.830534934997559, 62.88337326049805, -20.878875732421875, -13.79117202758789, -13.019002914428711, -2.5745363235473633, 7.712985038757324, -18.34477996826172, 1.3570939302444458, -18.33639144897461, -14.626672744750977, -21.671939849853516, 10.173983573913574, 5.564438343048096, -73.5653305053711, -47.962242126464844, 9.10100269317627, -10.337010383605957, 59.394893646240234, -1.6840375661849976, -3.955976724624634, 9.972034454345703, 10.950456619262695, 10.034374237060547, -11.571080207824707, -1.0772384405136108, -35.48460388183594, 43.30226135253906, 33.46714401245117, 0.030037088319659233, -49.677894592285156, -21.648719787597656, 47.86350631713867, 40.023738861083984, -33.03044128417969, -10.461617469787598, 28.05082130432129, -62.85028839111328, 0.49693089723587036, -34.89893341064453, -37.25096130371094, 30.370723724365234, -16.886974334716797, -6.407612323760986, 43.681400299072266, 75.52433013916016, -42.91513442993164, 33.44715881347656, -39.54875564575195, -39.281333923339844, 0.03302003815770149, 20.297016143798828, 23.643163681030273, -15.268136978149414, 8.905202865600586, -26.824684143066406, -22.029273986816406, 1.4367396831512451, 9.554043769836426, -43.377079010009766, -44.96906280517578, -21.877328872680664, -6.0349040031433105, 45.909332275390625, 34.181182861328125, -10.68407154083252, -9.976438522338867, 28.889917373657227, 4.389102458953857, -2.867762327194214, 44.65272521972656, -17.585058212280273, 49.3366584777832, 58.83368682861328, 42.53459167480469, 45.242576599121094, -44.67130661010742, 6.944151401519775, -26.184558868408203, -2.2517518997192383, -34.26230239868164, 8.13123607635498, 28.27305793762207, -12.1121244430542, 10.236995697021484, -52.80192565917969, -43.62022399902344, -19.152265548706055, -23.828899383544922, 30.78168487548828, -1.9763432741165161, -8.419177055358887, -21.348731994628906, 24.287485122680664, -3.501556158065796, -17.644651412963867, 10.401710510253906, 4.815659999847412, 17.678895950317383, -44.750328063964844, 26.891300201416016, -20.481725692749023, -0.7350184321403503, -29.68215560913086, -50.61819076538086, -19.783157348632812, -9.256244659423828, -0.4221619665622711, -15.121284484863281, 33.6744499206543, 38.87355041503906, 34.57341766357422, 34.04648208618164, 16.437397003173828, -38.8045539855957, 11.031610488891602, 38.258846282958984, -7.49373722076416, 39.90837097167969, 8.722880363464355, 9.934657096862793, 26.352523803710938, -58.87562561035156, -46.38298797607422, 7.9524383544921875, -41.10774230957031, 37.60881423950195, 3.8876049518585205, -53.760929107666016, 44.17781448364258, -6.031604766845703, 15.348576545715332, 45.640724182128906, -8.731175422668457, 37.26328659057617, -13.61270809173584, 19.583572387695312, 15.405916213989258, 43.79265213012695, 13.9072847366333, -16.053152084350586, 51.80548858642578, 30.978899002075195, 10.36187744140625, 39.15312957763672, 56.15806579589844, -30.76622772216797, -1.0612977743148804, -29.538862228393555, -31.809171676635742, -15.085344314575195, -3.7710602283477783, 19.798200607299805, -21.22902488708496, 42.08121871948242, 12.63531494140625, 39.85772705078125, -23.404050827026367, -5.731405258178711, -3.7110202312469482, -12.868002891540527, 36.97740173339844, -14.118108749389648, 68.51107788085938, -7.310083389282227, 4.469163417816162, -19.78699493408203, -20.75296974182129, 61.365760803222656, -38.77293395996094, 0.46217283606529236, 14.28654956817627, -28.814374923706055, -19.8515625, -8.050731658935547, -24.278200149536133, -20.826528549194336, -5.343977928161621, 49.289894104003906, -11.878962516784668, 40.171363830566406, -22.99260902404785, 11.407415390014648, -29.110368728637695, 5.559933662414551, 31.41754722595215, 22.364336013793945, 7.155996322631836, -54.68602752685547, -12.926186561584473, 13.590893745422363, -44.60561752319336, -20.956384658813477, -34.95451354980469, -20.58156394958496, -14.144425392150879, 18.617746353149414, 44.97096633911133, -11.058093070983887, -23.769359588623047, 37.690345764160156, 3.255417823791504, -73.79431915283203, 9.05107307434082, -6.201327800750732, 50.27496337890625, -43.840126037597656, -4.23590087890625, 3.117840528488159, -21.044546127319336, -4.596301555633545, -3.7300808429718018, -4.5424299240112305, 24.954912185668945, -24.065452575683594, 54.67763137817383, 23.338565826416016, -12.744277954101562, -68.5975570678711, -13.832874298095703, -54.69628143310547, -2.297377586364746, 5.09321403503418, 22.902822494506836, -22.186004638671875, 4.3941450119018555, -23.544376373291016, -5.293761253356934, -6.147963523864746, -7.971370697021484, 25.567745208740234, 4.660480499267578, -1.1088675260543823, 13.68090534210205, -1.2947285175323486, 38.91244888305664, -36.64975357055664, -0.4972842335700989, 22.55963706970215, 14.300115585327148, 10.648674011230469, 25.673080444335938, 8.675982475280762, 39.43205642700195, 33.30767059326172, 24.787412643432617, 36.65574264526367, -58.872318267822266, 16.954652786254883, 38.536712646484375, 9.046976089477539, -13.800875663757324, -63.21294403076172, 4.271960258483887, -3.3334977626800537, 25.528034210205078, 28.67057991027832, -11.069931983947754, 67.42943572998047, 21.559831619262695, -0.12154392153024673, -29.36707305908203, 5.821905136108398, 6.780404567718506, -12.169098854064941, -24.753299713134766, 39.92931365966797, -62.25940704345703, 17.252721786499023, 60.747657775878906, -17.0203857421875, -0.9103468060493469, -44.85968017578125, 5.333374977111816, -9.054903984069824, -15.1138916015625, -14.218073844909668, -4.609103202819824, -6.401588439941406, -54.339351654052734, -36.1110954284668, -39.75302505493164, 23.282819747924805, 31.932069778442383, 19.283842086791992, -32.79897689819336, -20.954458236694336, 33.672115325927734, -67.8007583618164, 28.356792449951172, -37.80192947387695, 36.36500930786133, 17.9813175201416, 41.270015716552734, 5.811059474945068, 59.408935546875, 24.503448486328125, -30.692039489746094, -50.9205436706543, -6.944115161895752, 7.863707542419434, 26.363649368286133, 30.495460510253906, 12.052804946899414, 51.95158386230469, -2.4870312213897705, 30.728788375854492, 39.76129150390625, 19.541370391845703, -18.167633056640625, 16.09758758544922, 16.687803268432617, -17.878353118896484, 10.55062484741211, 5.625921249389648, -15.910974502563477, 21.37586784362793, -15.727715492248535, -10.143332481384277, -15.10708236694336, -34.47340393066406, -4.709228992462158, 32.43191909790039, -60.07276153564453, -55.25442886352539, -59.34346008300781, -25.206010818481445, -26.32758331298828, -18.570083618164062, -68.02360534667969, 28.105451583862305, -8.849596977233887, -10.217263221740723, -15.065975189208984, 41.447227478027344, 10.802974700927734, 55.95613479614258, 2.8396084308624268, -19.057209014892578, -24.02501106262207, -25.77574920654297, 13.727215766906738, -27.572233200073242, -13.252372741699219, -3.5864343643188477, -22.826457977294922, 4.525550842285156, 42.04501724243164, -39.73030090332031, -28.269338607788086, 36.8073616027832, 9.333540916442871, -27.079050064086914, 28.47881507873535, -16.22148323059082, 31.128803253173828, -21.42390251159668, -36.95871353149414, -37.80418014526367, -51.954811096191406, 31.756378173828125, -48.0557746887207, -16.010095596313477, -3.9771034717559814, -1.7006229162216187, 11.501425743103027, -39.87902069091797, -48.38801574707031, -7.500333309173584, 30.7912654876709, 11.843917846679688, -22.946773529052734, -4.5817646980285645, 6.310292720794678, -26.311771392822266, 31.577436447143555, 52.31729507446289, 47.15676498413086, 6.36625337600708, -3.1645090579986572, -16.841047286987305, -34.1097297668457, -22.704130172729492, 61.69196701049805, 2.0610501766204834, -46.12007141113281, 49.54396057128906, 17.51771354675293, 0.39808228611946106, -28.026371002197266, 57.9432258605957, -18.146772384643555, 32.01844787597656, 17.188095092773438, -9.711051940917969, 5.383654594421387, -25.577869415283203, -34.58767318725586, 35.56231689453125, 30.793487548828125, 48.004093170166016, -26.364599227905273, 23.16546630859375, -13.299092292785645, 10.986985206604004, -34.52861785888672, 17.38298225402832, -0.4872376322746277, -33.54277038574219, -35.46181106567383, 0.11009565740823746, -45.41081619262695, -34.177574157714844, 41.52415466308594, -35.240013122558594], [4.893930435180664, 36.93634796142578, 0.7543736100196838, -46.13106918334961, -11.503494262695312, 27.589139938354492, 1.1058361530303955, 41.7450065612793, 11.553403854370117, -50.0418815612793, -24.671964645385742, -61.83578109741211, -8.628283500671387, -16.464807510375977, 13.189508438110352, -12.709487915039062, 4.428207874298096, -37.64851379394531, -94.63636016845703, 6.741878986358643, -28.09168243408203, 44.512107849121094, 15.864188194274902, -19.69014549255371, -11.903850555419922, 13.671894073486328, 2.5705013275146484, -29.510644912719727, -37.72382354736328, -2.918445587158203, -15.268631935119629, -17.983341217041016, -45.07820510864258, 36.20064163208008, -2.905306339263916, -10.765478134155273, 33.60399627685547, -18.064058303833008, 38.636390686035156, 37.55684280395508, -25.269102096557617, -3.2573657035827637, 3.803082227706909, 15.058293342590332, -43.303775787353516, 25.543508529663086, 0.8237804770469666, 13.411120414733887, -13.651412010192871, -18.284276962280273, -13.566720962524414, -16.145488739013672, -16.65628433227539, -8.308016777038574, -42.82330322265625, -15.491973876953125, 6.961759090423584, -6.2601094245910645, -27.13384246826172, 15.037878036499023, -17.207460403442383, 8.71934986114502, -27.892358779907227, -18.978343963623047, -17.987545013427734, 2.9979565143585205, -4.610012531280518, 43.93828201293945, -33.82143020629883, 11.671744346618652, -9.987020492553711, 19.34728240966797, 3.7377004623413086, -35.765602111816406, -20.215404510498047, 8.774116516113281, -68.15362548828125, 21.393409729003906, 1.8979458808898926, -46.868865966796875, -0.4215855896472931, -20.86269187927246, -50.28059768676758, -38.05120849609375, -28.445281982421875, -44.76920700073242, -10.276172637939453, -29.352678298950195, -8.106864929199219, -14.193265914916992, 21.648954391479492, 26.706090927124023, 0.14097994565963745, -12.212480545043945, -42.64996337890625, -29.654008865356445, -76.74352264404297, 19.981605529785156, 5.561695098876953, -20.566308975219727, -16.99474334716797, -0.09549061954021454, -7.285764694213867, 30.929161071777344, 7.559078216552734, -7.600992202758789, 21.33622932434082, 20.768232345581055, 15.301190376281738, 32.516387939453125, 24.415340423583984, -37.079166412353516, -2.6876842975616455, -20.527957916259766, -2.7226264476776123, 16.484657287597656, -15.150629043579102, 2.546797513961792, -74.22053527832031, 47.176109313964844, -14.756032943725586, -21.061613082885742, 32.6054573059082, 30.663814544677734, -20.210662841796875, 18.00851821899414, -32.51386642456055, -45.96474075317383, 16.670873641967773, 0.2213485985994339, -25.5679874420166, 48.662078857421875, 16.035898208618164, 22.820884704589844, -43.900115966796875, 5.172184467315674, -80.0120849609375, 48.686466217041016, 37.241966247558594, 3.5260798931121826, 17.280269622802734, 49.35609436035156, 2.242105484008789, -42.86227798461914, 20.908512115478516, -37.12345504760742, 17.418031692504883, 9.603931427001953, 10.11696720123291, -15.138916969299316, -20.520893096923828, 5.8872456550598145, -27.670997619628906, -12.383426666259766, -59.1142692565918, -25.589378356933594, 6.871674537658691, -15.726980209350586, 66.12493133544922, -52.83441162109375, 28.018198013305664, 86.54694366455078, 2.7762463092803955, -5.2293171882629395, 3.208454132080078, -20.337505340576172, -23.961374282836914, 53.59113693237305, -48.75752258300781, -7.838757514953613, -20.003061294555664, 52.01923751831055, 0.19936469197273254, 26.730764389038086, 23.664470672607422, -3.7443792819976807, -14.061100006103516, -6.369776725769043, -29.38737678527832, -11.77029800415039, -0.03808863088488579, -40.288169860839844, 26.39678192138672, 3.6150002479553223, -30.35219955444336, -6.752678871154785, 16.786958694458008, 13.003254890441895, 29.10871696472168, 38.57491683959961, 3.836071729660034, 10.047731399536133, -6.621502876281738, 27.49256134033203, -34.127357482910156, 14.619322776794434, -5.390780448913574, -30.69658660888672, 10.645648956298828, -11.963345527648926, 9.726943016052246, 6.5238800048828125, -36.06806182861328, 3.064396858215332, -0.3050537109375, 53.43299865722656, 8.446758270263672, -20.562156677246094, 9.858675003051758, -18.200305938720703, -2.1203088760375977, 51.305091857910156, 102.6569595336914, -1.1609145402908325, -42.84915542602539, -12.129401206970215, 29.521312713623047, -32.56165313720703, 26.683273315429688, -24.02651023864746, -34.583675384521484, 1.4015403985977173, 15.30810546875, 42.46055603027344, -35.865257263183594, -11.601621627807617, -22.616220474243164, 19.725448608398438, 28.236894607543945, -4.933104515075684, -29.9147891998291, 19.54018783569336, 6.794433116912842, 1.6240534782409668, -23.141550064086914, -1.9059184789657593, 39.18854904174805, -13.25146484375, -2.6311075687408447, -17.221899032592773, 3.4214770793914795, -1.0656758546829224, -39.848838806152344, -83.3342056274414, 36.60298156738281, -29.8718204498291, -31.531383514404297, 38.7763786315918, -2.5561373233795166, 30.511465072631836, 4.83613920211792, -22.17475700378418, -4.859086036682129, 24.630882263183594, -38.69609451293945, 28.014785766601562, 44.373199462890625, 30.135997772216797, 27.056621551513672, 38.019798278808594, 11.222475051879883, 46.88923645019531, -13.555635452270508, -23.646705627441406, 3.2537262439727783, 39.638675689697266, 10.008889198303223, 5.3272271156311035, 15.210587501525879, 19.005334854125977, 22.049924850463867, -20.13597297668457, -16.645151138305664, -41.370704650878906, -19.805469512939453, 4.876772880554199, -9.948896408081055, -7.847532749176025, 46.82318878173828, 2.4195144176483154, -18.88384437561035, 31.973730087280273, 30.355567932128906, -35.976314544677734, 17.371732711791992, 6.496077060699463, 7.910654067993164, 9.008919715881348, -29.94796371459961, -7.277136325836182, -40.603694915771484, 3.8202145099639893, -15.506072044372559, -9.03090763092041, 10.433902740478516, -33.13126754760742, 74.10993194580078, 32.37504959106445, 20.110984802246094, -45.374515533447266, 14.8306245803833, 73.83230590820312, -58.53487777709961, 37.10310745239258, 10.301403045654297, 15.512738227844238, 56.874324798583984, -34.96278762817383, 23.098859786987305, -45.79867935180664, -3.9528050422668457, -9.925422668457031, 42.90593338012695, -22.417823791503906, 32.065372467041016, -33.755062103271484, -42.652984619140625, -24.061067581176758, 12.315471649169922, 59.922767639160156, -9.267138481140137, -35.24576950073242, -18.008609771728516, -9.223915100097656, 16.23011016845703, -39.46574020385742, -8.88402271270752, -11.09774398803711, -37.779388427734375, 29.160280227661133, 14.054587364196777, 17.383974075317383, -7.803825378417969, -2.172821044921875, 11.24306869506836, 1.5926259756088257, 28.546833038330078, 13.468301773071289, 44.64425277709961, 4.164925575256348, -23.298702239990234, -115.76429748535156, -47.26432418823242, -44.59074783325195, 9.710168838500977, -11.400726318359375, 11.885522842407227, 8.441360473632812, 29.30082130432129, 30.18572998046875, 1.9362518787384033, -6.795273780822754, 1.6650899648666382, 27.67583465576172, -3.5238192081451416, 25.667720794677734, 41.605072021484375, -13.050514221191406, 42.78765869140625, 34.65367126464844, -17.219099044799805, -6.636661529541016, -29.50731658935547, -51.40793228149414, 27.143964767456055, 5.492527008056641, 13.215206146240234, 15.295177459716797, 14.89828109741211, -30.29238510131836, 6.937079906463623, -0.3324928879737854, 2.9460651874542236, 19.113853454589844, 8.855988502502441, -12.417675018310547, 0.9001857042312622, 34.25627517700195, -2.566826343536377, 17.725120544433594, -6.147567272186279, 14.275850296020508, -37.28512954711914, 35.30415725708008, -4.801877975463867, -4.774188041687012, -17.43256950378418, 4.550177097320557, -20.633615493774414, -42.3142204284668, 30.230587005615234, 25.422584533691406, -10.728034019470215, -46.649070739746094, 62.956974029541016, -6.364196300506592, -14.979730606079102, -11.180938720703125, -0.28758928179740906, 66.6996841430664, 12.468828201293945, -7.364147663116455, 6.139030933380127, 5.228487491607666, -35.47044372558594, 11.974319458007812, 27.50766372680664, 60.04750061035156, -26.613399505615234, 16.996173858642578, 23.804336547851562, -3.0321733951568604, -30.941444396972656, 37.9429817199707, 3.0545992851257324, 24.175586700439453, -21.224895477294922, 39.2041015625, 28.291553497314453, -28.360445022583008, -20.785438537597656, -19.102214813232422, 20.54098129272461, -13.990464210510254, -41.73722457885742, -34.398658752441406, -10.445637702941895, -0.46063730120658875, 1.7718439102172852, 46.34453201293945, 51.837562561035156, -9.19560718536377, 6.159424304962158, 2.0996077060699463, 13.819005012512207, 21.864118576049805, -1.944866418838501, 61.419403076171875, 6.035780906677246, -52.10140609741211, -13.282912254333496, 8.964141845703125, 15.754439353942871, -48.67348098754883, 21.591381072998047, 14.917763710021973, 4.341677665710449, 19.53178596496582, 37.518699645996094, -7.807117462158203, 36.29669189453125, -14.944326400756836, 13.825329780578613, -17.475234985351562, 17.26944351196289, 8.722790718078613, 0.25744980573654175, -12.82948112487793, 57.26689529418945, 0.5773416757583618, -10.19472599029541, -45.435001373291016, 17.85291862487793, -0.8661614656448364, 55.90658950805664, 30.791208267211914, -41.311405181884766, 35.821231842041016, -33.42930603027344, 25.93903923034668, 0.37004193663597107, 1.3518486022949219, -18.795814514160156, 13.684114456176758, -34.82817077636719, -15.13676929473877, 22.410179138183594, -2.7202723026275635, -35.390865325927734, -2.126582145690918, -27.922937393188477, 32.25519561767578, -20.87824249267578, -11.093709945678711, 46.9918327331543, 1.385183334350586, 9.50371265411377, 8.330780029296875, 41.61045837402344, 2.7280850410461426, 8.305495262145996, 20.89348030090332, 23.634464263916016, 19.76966094970703, -48.879119873046875, -3.177429676055908, 29.446033477783203, 21.415390014648438, -9.432474136352539, 32.20326614379883, -26.28102684020996, 26.389434814453125, 24.120662689208984, -9.155694961547852, 13.86983871459961, -51.15827560424805, -33.01774597167969, -3.029055118560791, -31.6574764251709, 15.79922103881836, 59.130836486816406, -4.824246883392334, 36.494449615478516, -35.921417236328125, 14.724675178527832, 53.3851432800293, 71.9700927734375, -18.17723846435547, -47.27071762084961, -11.035218238830566, -5.229800224304199, 5.430761337280273, 24.87824058532715, 31.15642547607422, 24.690624237060547, -14.662264823913574, 65.56248474121094, -14.892783164978027, -17.71982192993164, -41.46149444580078, -5.191370487213135, -5.499508857727051, 34.108787536621094, -23.420133590698242, 18.630502700805664, 1.91154146194458, 6.807393550872803, 3.5198888778686523, 6.451244354248047, -13.689360618591309, -95.38015747070312, 14.735657691955566, 1.0622022151947021, 2.304361343383789, -19.384540557861328, 5.499819755554199, -22.669782638549805, 41.16081237792969, -9.650739669799805, -44.569828033447266, -15.0426025390625, -22.308385848999023, 42.85179901123047, -40.309539794921875, 8.749675750732422, 6.631771564483643, 23.704431533813477, 45.664894104003906, 7.2208733558654785, 47.56584167480469, 23.322704315185547, -25.611976623535156, -28.81332015991211, 40.11452102661133, -5.091453552246094, -67.29813385009766, 33.09794616699219, 76.14879608154297, 9.069685935974121, 17.06294822692871, -26.050790786743164, 4.386745452880859, 23.9294490814209, 1.5169607400894165, -23.88583755493164, -10.608549118041992, -42.211387634277344, 53.12672424316406, -32.55638885498047, -105.55017852783203, 0.6050785183906555, 12.764066696166992, -33.344078063964844, 31.490005493164062, -23.850666046142578, 41.39164733886719, 1.569692611694336, 49.18714904785156, 1.8649439811706543, -28.10672950744629, -55.62794494628906, -21.861705780029297, 2.163907766342163, -13.09370231628418, -49.36663818359375, 0.19811083376407623, 18.821701049804688, -37.093544006347656, -10.364605903625488, 14.220389366149902, -18.572935104370117, -16.541101455688477, 3.822178602218628, -49.60868453979492, -30.240215301513672, -38.48865509033203, -1.18623948097229, 5.763637065887451, 22.833681106567383, 15.63419246673584, -2.5281167030334473, -14.547628402709961, -36.19329833984375, -61.49422073364258, 14.398499488830566, 47.99962615966797, -0.4264529049396515, -4.557362079620361, -28.069143295288086, -71.00670623779297, 10.454123497009277, -9.872831344604492, -1.504881501197815, 27.728513717651367, -17.353134155273438, -18.8718318939209, 24.857803344726562, 4.393168926239014, 16.46493148803711, -22.336193084716797, -12.196084976196289, -38.6894645690918, -6.898586273193359, -15.58189582824707, -33.91658020019531, -58.93074035644531, -12.098158836364746, 7.603896141052246, 3.9745116233825684, -13.149579048156738, -2.492267608642578, 19.145612716674805, 9.407696723937988, 13.929821014404297, 2.7789320945739746, 48.8659782409668, -5.488554000854492, -2.0628230571746826, -17.401538848876953, -0.6121377944946289, -72.20662689208984, 23.009538650512695, -35.43602752685547, 87.53007507324219, -11.658459663391113, 24.19761085510254, -60.10799026489258, -16.464656829833984, 1.2933329343795776, -22.514890670776367, -2.982771635055542, 38.93482971191406, -53.89323425292969, -17.8931941986084, 0.8854066133499146, 12.5167236328125, 34.009159088134766, 21.107341766357422, 44.01380920410156, -33.95161056518555, -28.424579620361328, 20.155385971069336, 27.542133331298828, -19.812255859375, -31.066375732421875, 19.552383422851562, 3.395425796508789, -1.5581214427947998, -33.67592239379883, 0.08448732644319534, -17.84687614440918, 39.74716567993164, 12.1373929977417, 1.6181886196136475, -0.4163484275341034, -18.6070499420166, -15.728395462036133, 33.85501480102539, 10.696910858154297, 17.923158645629883, 16.970542907714844, 0.6894276738166809, -56.5631217956543, -1.4827027320861816, -28.197484970092773, 18.67897605895996, 3.192291259765625, 10.906526565551758, -3.838235378265381, 24.831754684448242, 12.479232788085938, 20.144847869873047, -29.024208068847656, 2.569695234298706, -17.057933807373047, 0.15195845067501068, -28.42449951171875, -27.6370849609375, -61.90764617919922, 28.747230529785156, 4.813381671905518, 9.01146411895752, -18.968891143798828, 22.241872787475586, -29.606645584106445, -14.88070297241211, 30.239856719970703, -38.34571075439453, 38.697540283203125, -62.8152961730957, 5.597213268280029, 43.11093521118164, 27.933841705322266, 34.40203094482422, 21.925559997558594, 14.515419006347656, -8.306364059448242, 0.4367341995239258, 10.027679443359375, -36.54725646972656, -20.59168243408203, -24.038320541381836, -13.40148639678955, -21.438343048095703, -20.865922927856445, -30.174654006958008, -39.129329681396484, 7.3555498123168945, -38.2210693359375, 45.33580017089844, -16.86148452758789, -0.5885083675384521, -11.394144058227539, 0.5525140762329102, -64.67405700683594, -30.054874420166016, 24.00029945373535, -7.380557060241699, -70.90819549560547, -10.440292358398438, -24.99846839904785, 13.166791915893555, 21.31733512878418, -35.44396209716797, 3.1780452728271484, -52.202327728271484, 37.11137390136719, -9.861799240112305, 33.93669891357422, 41.20783996582031, -14.418965339660645, -44.80113220214844, 15.323714256286621, 72.03691864013672, 39.51018524169922, -20.031110763549805, -7.15076208114624, -11.954934120178223, 12.647798538208008, 24.445528030395508, 5.6041460037231445, 6.533801555633545, 39.93404769897461, -18.606550216674805, -18.295799255371094, -3.7930145263671875, 6.144478797912598, -36.32231903076172, 4.44759464263916, -5.8308939933776855, -12.830306053161621, -48.230438232421875, -28.908096313476562, -20.716445922851562, 28.85455894470215, -63.09247970581055, 15.541487693786621, -6.316832542419434, -34.642982482910156, 9.401213645935059, -5.418665409088135, 43.29109191894531, 16.41292381286621, 29.048471450805664, -13.336055755615234, -16.558528900146484, 5.990047454833984, -39.29219055175781, -18.483409881591797, -3.70729923248291, 31.305341720581055, -25.63544273376465, 9.702987670898438, -8.834370613098145, -31.998849868774414, 42.58628845214844, 2.7696518898010254, 13.908703804016113, -22.421098709106445, -57.92026901245117, -7.994067192077637, 16.902692794799805, 10.177703857421875, 40.921913146972656, -29.646400451660156, 15.127156257629395, 16.967044830322266, -47.941261291503906, 6.61088228225708, -2.858177900314331, 8.415497779846191, -23.272842407226562, 3.1396663188934326, 33.39213180541992, 1.046743631362915, -5.868164539337158, 12.38453197479248, 11.932866096496582, -82.27104187011719, 47.55461120605469, -46.253028869628906, -32.83834457397461, 28.251638412475586, 7.6329545974731445, -45.584415435791016, -3.764836311340332, -27.18303680419922, -3.8869338035583496, -24.309110641479492, 14.276246070861816, -1.284808874130249, -4.375441551208496, -51.01536560058594, -10.895759582519531, 14.966888427734375, 25.32094955444336, -21.936416625976562, 34.810699462890625, 15.817188262939453, 53.16990661621094, -19.482929229736328, -15.033475875854492, -15.737861633300781, -5.418828964233398, 9.415055274963379, 32.069454193115234, -12.009648323059082, -11.758265495300293, -14.025991439819336, 35.57944869995117, -28.58111572265625, 17.925554275512695, 28.4879150390625, 5.137033939361572, -24.461225509643555, -25.937929153442383, -10.597867012023926, 24.982072830200195, -20.070720672607422, 37.91265869140625, -39.70927429199219, 14.8333101272583, 8.246118545532227, 18.499950408935547, 26.494731903076172, -31.115520477294922, 10.958502769470215, -22.208738327026367, 3.933763027191162, 9.766667366027832, 38.607383728027344, -11.300355911254883, -19.699054718017578, 9.674765586853027, 46.39959716796875, -31.906749725341797, -12.623211860656738, 13.833677291870117, 14.211836814880371, 8.147419929504395, -9.268390655517578, -2.468801498413086, 25.29691505432129, 6.249175071716309, -15.321934700012207, -8.863784790039062, 40.918060302734375, 31.967443466186523, -18.932193756103516, 28.810379028320312, -23.761760711669922, 12.165307998657227, -35.57660675048828, -11.489989280700684, 16.289682388305664, -63.489810943603516, 58.10739517211914, -12.930953025817871, 40.96464920043945, 10.017250061035156, -12.861661911010742, 45.220252990722656, 53.88004684448242, -0.4634188115596771, 33.745018005371094, 29.407695770263672, 14.52144718170166, 18.63763999938965, -5.6079301834106445, -9.30637264251709, 1.795658826828003, -52.39219665527344, -20.03598403930664, 31.395729064941406, 63.32069778442383, 13.444525718688965, -7.757264137268066, 8.5536470413208, 16.79048728942871, 6.504840850830078, 0.7109088897705078, 17.635169982910156, 14.66261100769043, -3.7807421684265137, -14.628664016723633, 15.101022720336914, -32.56013870239258, 23.542007446289062, 53.78096008300781, -22.850719451904297, 4.079463481903076, 43.3859748840332, 32.2926025390625, 4.436646938323975, -27.261730194091797, -32.810821533203125, 1.332388162612915, -5.578885555267334, -35.79559326171875, -38.47148513793945, 7.7888407707214355, -45.51510238647461, 43.763275146484375, -57.2884521484375, -27.78870391845703, -67.47301483154297, -3.8918213844299316, 57.564125061035156, 71.40557861328125, -50.78877639770508, 24.546611785888672, -39.454837799072266, -12.403539657592773, 46.9781379699707, -7.027254104614258, -28.670719146728516, -10.269559860229492, -6.025085926055908, 11.012046813964844, 30.822837829589844, -60.11928939819336, 22.113868713378906, 18.29338836669922, -18.18537712097168, 59.28902816772461, -4.214192867279053, 6.909143924713135, 53.74116134643555, 39.268489837646484, -92.86788177490234, -5.174100399017334, 0.7475578188896179, -0.523790180683136, -12.741987228393555, -0.899345874786377, 18.962417602539062, -43.79401779174805, 16.456941604614258, -49.67552185058594, -56.100914001464844, -23.229032516479492, 43.98442077636719, -6.228265285491943, 21.013347625732422, 29.707565307617188, 9.408044815063477, 49.743839263916016, -39.22580337524414, 44.40339660644531, 7.027993679046631, -26.389299392700195, -26.859086990356445, 14.018714904785156, -16.25609016418457, -51.971073150634766, 40.64043045043945, 0.46406710147857666, -89.87149810791016, -12.313369750976562, -14.70828914642334, 4.61190128326416, 12.954765319824219, 31.25729751586914, 51.2752685546875, 0.05397681146860123, 40.41272735595703, -6.676266193389893, -12.202789306640625, -53.190696716308594, 3.2244303226470947, 41.89003372192383, -55.995723724365234, -34.27955627441406, 40.709983825683594, 22.839372634887695, -18.628582000732422, -43.221900939941406, 12.421531677246094, 14.661153793334961, -20.906784057617188, -13.660322189331055, 2.040360927581787, -0.5708063244819641, -16.38773536682129, -25.329626083374023, 33.53745651245117, 46.298831939697266, -7.743773460388184, 47.9727897644043, -27.171802520751953, 17.10741424560547, 8.138591766357422, 50.269893646240234, 31.082304000854492, 11.480742454528809, 35.128143310546875, 5.918841361999512, 57.09352111816406, 8.602989196777344, -22.069278717041016, -3.0977623462677, 69.12101745605469, -48.21968078613281, 7.264830112457275, -3.1212568283081055, 43.06796646118164, 30.81963348388672, -36.53297424316406, 2.3752193450927734, 45.86093521118164, -43.65647888183594, 1.2087485790252686, 20.21177101135254, 8.11252212524414, 41.42107391357422, -34.04328155517578, 19.550758361816406, -12.015063285827637, -15.018707275390625, 32.4000244140625, 16.64695930480957, -20.462505340576172, 50.7347412109375, 5.299931526184082, -1.7539966106414795, 17.841373443603516, 27.717058181762695, -77.8520278930664, 11.316189765930176, -28.56181526184082, 18.75925064086914, 15.45962142944336, 12.832453727722168, 6.383580684661865, -56.56704330444336, -25.684873580932617, 29.912548065185547, -56.00444412231445, 21.67221450805664, 1.8649730682373047, -22.34549903869629, -3.6656153202056885, 31.92836570739746, 4.192512512207031, -17.450956344604492, 0.8183279633522034, -16.390043258666992, 55.6320686340332, -2.083498001098633, -60.411922454833984, 4.000075817108154, -23.675630569458008, -21.959110260009766, -9.947646141052246, 27.424013137817383, -60.02488327026367, 24.56013298034668, -23.290319442749023, -33.63096237182617, 7.940605640411377, 8.30109691619873, -11.819875717163086, -3.268350839614868, 40.46710205078125, -0.14681032299995422, -10.215218544006348, -10.387419700622559, -35.01390838623047, 21.247314453125, 24.549579620361328, 9.524675369262695, 8.815431594848633, -23.444665908813477, 8.01768970489502, 5.6647162437438965, 1.489135980606079, -33.530792236328125, 6.61829948425293, -15.902352333068848, 43.43269348144531, 20.995616912841797, 27.931964874267578, -42.872474670410156, -7.9860429763793945, 5.7275590896606445, 3.4507100582122803, -1.4724565744400024, -45.7109375, 14.451536178588867, -25.01725959777832, 0.5032697319984436, -41.8300666809082, 39.3392333984375, 50.076820373535156, -39.39521026611328, -20.44495964050293, -12.373931884765625, 2.5678985118865967, -26.87091064453125, 38.23317337036133, 40.115692138671875, 14.573843002319336, -65.0167465209961, 10.391852378845215, 48.32114028930664, -3.398380994796753, 68.24496459960938, -17.653940200805664, -58.648563385009766, -44.95222091674805, 18.892539978027344, -33.16998291015625, -41.42293930053711, -4.1396660804748535, -4.779060363769531, -7.244452476501465, 5.985621929168701, -16.750595092773438, 6.406655788421631, 34.41664505004883, 2.8768086433410645, 0.4128511846065521, -1.893052101135254, 40.6262321472168, -3.7104532718658447, 10.785918235778809, 39.83259963989258, -38.38003921508789, -27.036510467529297, -17.6956844329834, -49.06380844116211, -12.619464874267578, 62.55197525024414, -19.21397590637207, 3.073941469192505, -29.524932861328125, 32.24347686767578, -33.3516845703125, -26.836414337158203, -25.568462371826172, 22.603431701660156, 22.826492309570312, 6.746755599975586, -5.135265350341797, -26.06378173828125, -10.134797096252441, 14.17264461517334, 65.46302032470703, 20.605487823486328, -31.447227478027344, 31.34391212463379, -34.924400329589844, -55.965232849121094, -16.85186767578125, -2.1351728439331055, -2.260023832321167, -6.385398864746094, 12.217473030090332, -2.10841703414917, -49.87751770019531, -4.997831344604492, -21.242706298828125, -19.453243255615234, -35.098182678222656, 6.755969047546387, -2.4416191577911377, 19.525100708007812, 33.45035171508789, -7.580395698547363, 4.957268238067627, -40.323570251464844, 3.420008897781372, -50.89952850341797, -15.232864379882812, -14.418817520141602, -26.195280075073242, 33.58123016357422, -57.03120422363281, -19.22663116455078, 38.44544219970703, 4.6834259033203125, -22.202068328857422, 8.23015022277832, 28.09982681274414, 21.726512908935547, 18.526294708251953, 57.260501861572266, -7.792455673217773, 4.156276702880859, -0.7197284698486328, -7.127078056335449, -63.778079986572266, -9.614819526672363, -4.507299423217773, -10.824735641479492, 17.304462432861328, -16.62179946899414, 8.032912254333496, -51.591094970703125, 4.318471908569336, -32.436466217041016, 5.11087703704834, 12.176773071289062, -70.5829086303711, -48.342647552490234, 21.1500301361084, -13.235179901123047, -12.88998031616211, -30.815956115722656, 11.151556968688965, -34.56740951538086, 13.262608528137207, 13.005436897277832, -52.223182678222656, -10.110872268676758, 6.344511985778809, -14.487951278686523, 2.3188230991363525, 9.261260032653809, -20.4556827545166, -9.780667304992676, -5.7537994384765625, -1.7988650798797607, -62.21171951293945, -5.588367938995361, 19.112165451049805, 18.368165969848633, -2.7951302528381348, -22.44084930419922, -7.159955024719238, -33.96500015258789, 17.1016788482666, -42.003360748291016, 4.218262672424316, 14.226319313049316, 4.702297687530518, -4.375487327575684, 3.9923157691955566, 20.89146614074707, -51.56037902832031, 0.2962568700313568, 9.68224811553955, 28.897151947021484, 82.93033599853516, -15.487497329711914, -63.19740676879883, 14.720833778381348, 0.9450139999389648, 54.83710479736328, 21.104846954345703, 40.74941635131836, 11.272913932800293, 23.802696228027344, 18.649261474609375, 5.761745452880859, -29.888811111450195, 44.82533264160156, 3.6255669593811035, -19.16534423828125, -7.229245662689209, -77.19219970703125, -4.448095798492432, 5.143476486206055, 99.50048828125, 9.233272552490234, -4.566827774047852, 41.31066131591797, -35.450592041015625, 21.318418502807617, -60.26773452758789, -72.44126892089844, -11.983487129211426, -3.4675235748291016, 42.60627365112305, 15.700206756591797, -24.88556480407715, -0.18044927716255188, -53.44794845581055, 0.15563298761844635, -26.61064910888672, 8.786983489990234, 23.694778442382812, 41.27641677856445, 6.6394147872924805, -64.34050750732422, 8.579872131347656, -0.9440217018127441, 19.366418838500977, 48.30146026611328, -7.398827075958252, -18.055850982666016, -44.45962142944336, 43.27292251586914, -81.89617156982422, -10.96153450012207, -19.987640380859375, 5.352293491363525, -10.35494613647461, -11.460641860961914, 7.266432762145996, 23.58003807067871, -19.51874542236328, -25.279455184936523, 9.23116397857666, -0.6769454479217529, 15.682997703552246, 47.18116760253906, 74.83008575439453, -24.53557586669922, 8.597795486450195, 15.021036148071289, -34.54435348510742, -15.71604061126709, -33.91316223144531, 39.48970031738281, 18.255401611328125, 24.201805114746094, 4.23529052734375, 29.53574562072754, 7.153957843780518, -12.51801586151123, 13.721850395202637, 17.695112228393555, 9.948058128356934, -32.17361831665039, 89.25662994384766, 21.625585556030273, 37.67286682128906, -14.793478965759277, 42.31643295288086, 13.803568840026855, -37.71868896484375, -4.43167781829834, 0.48830392956733704, -23.937698364257812, 30.06696319580078, -33.72256851196289, 21.145002365112305, -3.0230746269226074, 7.398971080780029, -40.030216217041016, 33.17962646484375, 12.41750717163086, -51.9475212097168, -21.527938842773438, 21.107145309448242, -3.5370121002197266, 9.918830871582031, 32.727439880371094, -14.016822814941406, 30.36139678955078, 36.82038497924805, 19.888286590576172, -45.32829284667969, 5.54005241394043, -64.34297180175781, -6.537888050079346, 3.958130121231079, 32.212440490722656, 1.9041492938995361, -18.152545928955078, -5.125608444213867, 2.5487992763519287, -8.34234619140625, 11.956247329711914, -16.985803604125977, 8.194721221923828, -7.945756435394287, 43.29404067993164, 5.334672927856445, -8.930838584899902, -57.01597595214844, 3.4448859691619873, 36.71924591064453, -45.948944091796875, 30.931278228759766, 5.630035400390625, 11.825971603393555, -34.69197082519531, -3.1062822341918945, -37.41524124145508, -12.354707717895508, -10.31835651397705, 66.74771118164062, -23.828407287597656, -42.488407135009766, -33.60124588012695, 14.888459205627441, 34.04621124267578, 23.237741470336914, -9.47430419921875, 2.995908260345459, -11.91221809387207, -23.74066925048828, -20.430877685546875, 48.97038269042969, -12.813002586364746, -16.166427612304688, 70.53604125976562, -39.192665100097656, -31.345163345336914, -41.37606430053711, 20.67483901977539, 8.88045597076416, 61.97795486450195, 9.188553810119629, 59.63329315185547, -34.53810119628906, 35.33699035644531, -35.55735778808594, -0.8350701332092285, -25.807828903198242, 23.746858596801758, 24.694387435913086, 17.384443283081055, 25.53089141845703, -42.51527786254883, 59.54606246948242, -35.971614837646484, 2.2038588523864746, -10.058486938476562, 48.00584030151367, 1.8402199745178223, -54.2066535949707, -4.76777982711792, 25.30057144165039, -19.407855987548828, -12.912699699401855, -8.488076210021973, -6.783634662628174, 10.559833526611328, 38.82331085205078, 8.217007637023926, -2.254228115081787, -37.61579895019531, -15.525525093078613, 39.07826232910156, 19.159881591796875, -24.539892196655273, 26.8024959564209, 8.126264572143555, 42.41506576538086, 3.9307827949523926, 30.855749130249023, -32.046932220458984, 17.62847900390625, 17.600074768066406, 11.641942024230957, 36.624420166015625, -39.31315231323242, 30.043888092041016, -11.2396879196167, 13.968124389648438, 71.5450668334961, -6.311105728149414, -23.337146759033203, 34.101097106933594, -56.40608215332031, -21.81511878967285, 18.84311294555664, 116.85540008544922, -43.245765686035156, 2.2154459953308105, -33.088226318359375, 33.50184631347656, 10.44697380065918, -15.972184181213379, -29.068204879760742, 10.80920696258545, 14.616629600524902, -12.017523765563965, 31.488086700439453, -17.053388595581055, 11.322928428649902, 43.686851501464844, 33.131141662597656, -9.369543075561523, 54.88374328613281, 47.27190399169922, -0.19364261627197266, -59.411346435546875, -37.140899658203125, 5.6208930015563965, -13.64619255065918, 9.705429077148438, -2.4737956523895264, -44.59794616699219, -25.154754638671875, -10.778647422790527, -15.862095832824707, -29.37166976928711, -46.5682373046875, 12.276741981506348, -38.77028274536133, 60.30828857421875, 66.47065734863281, 32.032135009765625, -28.706031799316406, -14.279077529907227, -24.0836124420166, 1.4334315061569214, 34.5110969543457, -19.82495880126953, 44.7872314453125, -14.940349578857422, 5.248264789581299, -35.278934478759766, 10.93545150756836, -8.332921981811523, -42.08802032470703, 11.959538459777832, 20.253910064697266, -19.69038963317871, -21.07346534729004, 49.245601654052734, -15.254897117614746, -10.820775032043457, 3.5696866512298584, 23.586734771728516, -2.907435894012451, -34.295265197753906, -15.00035572052002, 33.880977630615234, 19.680036544799805, -9.321270942687988, 53.48655700683594, -0.208951935172081, 4.274160861968994, -31.459449768066406, -36.373165130615234, 28.800277709960938, 16.12604331970215, -37.55278778076172, 30.394603729248047, 43.419830322265625, 1.793721079826355, 44.65160369873047, 2.698056221008301, 11.849488258361816, -12.507229804992676, -8.67026424407959, 35.78553771972656, 34.355491638183594, 11.142877578735352, -34.27621078491211, 13.289033889770508, -21.415071487426758, 47.86738586425781, -10.645994186401367, 16.019807815551758, 3.7431399822235107, 35.92734909057617, -71.17354583740234, 7.330674648284912, -32.87540054321289, 8.7849702835083, 46.94536590576172, -36.029563903808594, 26.440044403076172, -8.321444511413574, -15.015392303466797, 20.60232162475586, 6.354755878448486, -3.6967575550079346, 59.47865295410156, -34.37266159057617, -6.8578338623046875, 108.14054107666016, -43.12153244018555, 23.171382904052734, -18.27698516845703, 2.0252089500427246, -9.28542423248291, 18.735593795776367, -10.674498558044434, -4.1182074546813965, 42.652034759521484, -36.31285095214844, 61.363529205322266, -17.411846160888672, -3.1910440921783447, -12.656481742858887, 44.473228454589844, 21.1649227142334, -65.77216339111328, 16.324193954467773, -57.4886474609375, -16.63386344909668, 39.92913818359375, 5.402101516723633, 67.55172729492188, 74.7314224243164, -32.415184020996094, 9.627497673034668, -28.006324768066406, -12.522493362426758, -3.369645833969116, 14.659659385681152, 35.267982482910156, 13.178102493286133, 25.337831497192383, -22.19552230834961, -16.677465438842773, -19.09332275390625, 31.420398712158203, 29.799577713012695, 26.466278076171875, 1.4847359657287598, -4.379844665527344, 7.575058937072754, -12.549779891967773, -20.474323272705078, 44.91942596435547, -7.825769424438477, 26.506298065185547, 5.8168792724609375, 31.79456901550293, -15.08211898803711, -38.73158264160156, 35.87599563598633, -17.746902465820312, -33.5443115234375, -14.10024356842041, -24.157949447631836, 12.444747924804688, 25.982341766357422, -42.03863525390625, -8.76966381072998, -41.68320083618164, -58.77473068237305, 21.15135955810547, 79.71424102783203, 39.3810920715332, 11.515682220458984, -10.103461265563965, 13.590727806091309, 13.1199369430542, 24.456384658813477, -1.0406746864318848, -0.41196367144584656, -14.73158073425293, 44.173301696777344, 27.268308639526367, 8.987740516662598, 35.375858306884766, 44.762977600097656, 20.31202507019043, 14.999363899230957, -25.492517471313477, 29.41082763671875, 1.0282857418060303, -12.030014038085938, -54.863407135009766, 22.770009994506836, -41.38091278076172, -48.737327575683594, 8.589967727661133, -44.923431396484375, -37.25380325317383, 6.365207672119141, -20.68338394165039, -28.82470703125, -35.550987243652344, 23.99990463256836, -18.603113174438477, -33.68964767456055, 21.376646041870117, 6.670926094055176, -8.687423706054688, 30.079010009765625, 23.523170471191406, 0.9024040699005127, 9.21827220916748, 40.178199768066406, -3.6775784492492676, -26.8102970123291, 22.299911499023438, -5.053610801696777, 3.6394550800323486, 5.5488972663879395, -19.6473445892334, 9.056536674499512, -12.803574562072754, -7.0726728439331055, 10.768482208251953, 18.077322006225586, -19.47943115234375, -25.689449310302734, 33.42695999145508, 24.595462799072266, -1.2972710132598877, -49.56847381591797, 19.23300552368164, -12.182808876037598, -27.134117126464844, 6.744729995727539, -81.92684936523438, 33.138771057128906, 27.233840942382812, 2.4188668727874756, 12.352744102478027, -26.2009334564209, -37.092689514160156, 2.4469847679138184, 22.935317993164062, -19.58623504638672, -0.9030227065086365, 37.96343994140625, 64.27821350097656, 40.743690490722656, 3.4845259189605713, 23.633739471435547, -9.860122680664062, 10.376803398132324, 40.40320587158203, 40.156944274902344, 6.299680709838867, 9.651268005371094, 5.087080001831055, -23.77688980102539, 15.727852821350098, -3.1894242763519287, -32.48127365112305, -49.125205993652344, 14.486444473266602, -24.105937957763672, -30.396541595458984, 38.94342803955078, 5.621301651000977, 26.44573402404785, -21.48338508605957, 16.445772171020508, 30.591270446777344, -14.142600059509277, 0.758037805557251, -36.751644134521484, -46.96249008178711, -8.363100051879883, -13.946368217468262, 35.95905303955078, 10.75294303894043, 35.67401885986328, 6.5831522941589355, 15.450531005859375, 18.896610260009766, 11.900457382202148, -1.5684430599212646, 18.6739559173584, -8.506664276123047, -28.24599838256836, 15.260708808898926, 8.172236442565918, -3.181835889816284, -39.002010345458984, 43.818641662597656, -13.86570930480957, 9.342632293701172, 2.184713363647461, -10.101828575134277, -20.663715362548828, 23.988597869873047, -40.663734436035156, -15.14504623413086, -60.62542724609375, 3.5551795959472656, -32.02389907836914, -47.65088653564453, 11.940934181213379, -47.142147064208984, 12.9835844039917, -62.42731475830078, -6.153375148773193, -10.796875953674316, 3.5136916637420654, 17.145858764648438, 25.392871856689453, -46.38893508911133, -30.71962547302246, -47.160789489746094, -18.09222984313965, -2.2235591411590576, 61.68098831176758, 11.1646728515625, 1.3555039167404175, -56.86215591430664, -24.234745025634766, 28.597885131835938, 31.678483963012695, 53.3011589050293, 8.831392288208008, -17.269508361816406, 19.70453643798828, -5.548659324645996, 34.078857421875, -32.64514923095703, -0.10227366536855698, -16.918418884277344, -15.875099182128906, 3.215557336807251, 9.750022888183594, -13.105813026428223, 38.29243087768555, -28.926841735839844, -21.299264907836914, -0.6677290797233582, 12.699051856994629, 4.646039962768555, 39.700782775878906, -20.822738647460938, -27.00177764892578, 26.867027282714844, 60.09504318237305, 78.74205017089844, 4.006701946258545, -9.312105178833008, -11.322111129760742, -4.044715404510498, -23.706501007080078, -20.49551773071289, -14.433465957641602, -7.8610520362854, -7.922597408294678, -60.15576934814453, -56.2732048034668, -8.29163932800293, -23.8339900970459, -32.40614318847656, 20.7738037109375, -8.752233505249023, 11.73713493347168, -1.246577262878418, -4.801510810852051, -18.254331588745117, 77.89712524414062, -45.39152145385742, -18.17086410522461, -71.0849838256836, 40.61893081665039, -36.35556411743164, -39.034976959228516, 11.479790687561035, -46.02183532714844, 7.1692047119140625, -9.976395606994629, -45.03001403808594, -22.726911544799805, 33.09436798095703, 27.303068161010742, -28.517946243286133, -44.604801177978516, 58.899757385253906, 21.6522274017334, 9.201830863952637, -29.931438446044922, -62.20335388183594, -3.3606228828430176, 3.746565341949463, 30.735008239746094, 12.859109878540039, 15.84350299835205, 39.24010467529297, -45.83451461791992, 57.588714599609375, 14.877775192260742, 3.6286990642547607, 15.588966369628906, -8.970152854919434, 10.085755348205566, 8.519070625305176, 13.395886421203613, -1.5277761220932007, -25.666364669799805, -49.3813591003418, 3.0457215309143066, -39.95267105102539, -9.435540199279785, -23.256168365478516, -33.2493782043457, -15.000337600708008, 6.946655750274658, 16.83370018005371, 22.016260147094727, -48.6765022277832, -28.16975212097168, 1.057347297668457, -35.39192581176758, -19.988983154296875, 35.03044509887695, -18.77667999267578, 16.632978439331055, -20.43288230895996, -37.870304107666016, -2.5263729095458984, -67.6109619140625, -14.468050003051758, 50.78221130371094, -6.589961528778076, -6.005110263824463, -2.5962395668029785, -27.500341415405273, 28.282564163208008, 5.7119855880737305, -2.2444937229156494, -31.72029685974121, -9.02940845489502, 51.50376892089844, -10.454476356506348, -27.18482208251953, 75.70864868164062, 3.0503523349761963, 15.143623352050781, 19.947162628173828, -17.273399353027344, 41.48600769042969, -15.634578704833984, -9.753971099853516, 0.3397150933742523, 41.42961120605469, 3.342041492462158, 23.443317413330078, 17.565710067749023, 28.601764678955078, -5.018097877502441, 34.59989547729492, 10.538066864013672, 11.55570125579834, -16.282272338867188, 29.708826065063477, 30.312910079956055, 8.395855903625488, 35.1962776184082, -8.318879127502441, -34.946319580078125, 36.25770568847656, 3.437289237976074, -39.328365325927734, -9.54198169708252, -27.142187118530273, 8.119729995727539, -28.094837188720703, -13.18056583404541, 49.54452896118164, 50.01282501220703, -5.459411144256592, -12.965575218200684, 13.744742393493652, 10.241915702819824, 1.9567948579788208, 63.71493148803711, -12.82990550994873, -0.9706354737281799, -14.84609603881836, 1.6293922662734985, 18.177949905395508, 4.130828857421875, -0.6101716160774231, -42.92509460449219, 0.5716670751571655, -36.56342697143555, 29.041730880737305, -33.09592819213867, -62.35091781616211, -0.6035439372062683, -1.9505540132522583, -13.449935913085938, 40.89790344238281, 32.524253845214844, -38.16899108886719, 29.230438232421875, -39.54985427856445, 5.881518363952637, 51.8058967590332, 21.207422256469727, -51.630287170410156, 19.859989166259766, -26.59866714477539, 12.330435752868652, 74.12493133544922, -54.720680236816406, 16.19994354248047, 1.0453929901123047, 26.53293800354004, 2.5263752937316895, 43.20825958251953, 8.348073959350586, 23.64376449584961, 17.499969482421875, 53.44768142700195, -25.76570701599121, -82.80435943603516, 2.3003177642822266, -12.716103553771973, -6.603939533233643, 23.140167236328125, -45.30243682861328, -18.50508689880371, -53.98359298706055, 18.010269165039062, -40.72710037231445, 13.858453750610352, -17.39947509765625, 45.548805236816406, -30.671972274780273, -0.8361126184463501, 19.49873924255371, -87.1382827758789, -6.648376941680908, -17.012239456176758, -46.15271759033203, 29.14409637451172, -24.821290969848633, -61.40410614013672, 9.55156421661377, -60.517616271972656, 10.909327507019043, 8.591873168945312, 7.486170291900635, -21.330310821533203, -22.452720642089844, -0.4341612458229065, -14.567681312561035, -33.10503387451172, -14.844352722167969, -54.72142791748047, 7.214815616607666, -5.467006683349609, -6.366547107696533, -14.607318878173828, -30.958646774291992, 21.67107582092285, 1.336230754852295, -61.574920654296875, -22.389766693115234, 24.1660099029541, -45.19026184082031, -17.29073143005371, -68.8125, -61.75035095214844, -39.8656005859375, -6.495602607727051, -9.368489265441895, 43.472023010253906, -64.13095092773438, -38.25165557861328, -30.3377685546875, -12.438962936401367, 19.17971420288086, -96.53314971923828, 10.673476219177246, 0.4389897584915161, -28.60759925842285, 14.58535099029541, -52.59065246582031, 41.197021484375, 23.777233123779297, -14.715749740600586, -49.184226989746094, 57.85103225708008, 26.36037826538086, -2.6647448539733887, 26.99370765686035, 23.878408432006836, -26.138198852539062, -20.31574249267578, 5.285067558288574, -8.411603927612305, -27.19491195678711, 49.03814697265625, -9.714431762695312, -26.55803680419922, -0.37514397501945496, -63.8533821105957, -0.9722726941108704, 26.381629943847656, 43.337894439697266, 29.15785026550293, -0.09138111025094986, 0.5693140625953674, 62.42774200439453, -7.525071620941162, 11.35489559173584, -11.805337905883789, 5.569169044494629, 14.655508041381836, -14.711710929870605, -31.24191665649414, -42.9734992980957, -4.242515563964844, 73.03942108154297, -3.0017950534820557, -8.7053804397583, 40.16027069091797, 6.960690975189209, 18.34362030029297, -24.574785232543945, -40.493839263916016, 15.481828689575195, 53.54419708251953, -35.03984069824219, 7.299392223358154, -0.9700110554695129, 24.531147003173828, -28.610570907592773, 26.039091110229492, 22.20311164855957, 24.138303756713867, -21.586498260498047, -14.594573020935059, 9.490117073059082, 20.710615158081055, -24.590173721313477, -41.0760612487793, 35.239418029785156, 19.093788146972656, -30.22589874267578, -5.224310874938965, 49.686988830566406, -5.59193754196167, 28.298009872436523, -40.99747848510742, -30.28046226501465, 6.644796371459961, -14.304401397705078, 27.4152774810791, 14.283548355102539, 5.555618762969971, 3.140558958053589, 37.72493362426758, 25.348651885986328, -16.924163818359375, -22.612030029296875, -35.00593948364258, 39.08968734741211, -11.559526443481445, -22.596675872802734, -19.028846740722656, -12.108412742614746, -67.53813934326172, 27.37925910949707, -3.4653067588806152, -1.9099724292755127, 29.600732803344727, 16.01682472229004, -7.6479902267456055, -34.17122268676758, -24.45965576171875, -25.61933135986328, 41.65747833251953, -9.765427589416504, -5.912678241729736, -20.59977912902832, -41.39469909667969, 32.966434478759766, -19.213972091674805, -23.021663665771484, -31.41835594177246, 12.442294120788574, -20.662641525268555, 6.657526969909668, 6.232133388519287, -24.658599853515625, -28.876197814941406, 7.280602931976318, 85.11328125, -32.043983459472656, 41.3718147277832, 13.116203308105469, -30.207979202270508, -19.34746551513672, 7.383747577667236, 4.430425643920898, 0.24555173516273499, 25.84028434753418, -46.0488395690918, -23.255634307861328, 50.53364181518555, -24.092317581176758, 8.953184127807617, -57.90875244140625, -27.032373428344727, -24.512779235839844, -20.30418586730957, 0.36041152477264404, 20.29865264892578, 42.37655258178711, 8.632169723510742, -10.413209915161133, 5.720895290374756, 3.5656721591949463, 21.752216339111328, 14.10872745513916, 52.30486297607422, -8.569376945495605, 0.9847667813301086, 2.4044530391693115, 2.492220401763916, 7.607198238372803, -29.633405685424805, 0.6613121032714844, 27.526172637939453, 13.931532859802246, -13.675810813903809, 46.14349365234375, -38.50464630126953, -27.915672302246094, 59.78948974609375, 33.79175567626953, -42.351104736328125, 35.64534378051758, 20.142873764038086, -54.264923095703125, 34.27227783203125, 8.468125343322754, 5.465587139129639, 28.34585189819336, 74.20919799804688, -26.06447410583496, -37.87976837158203, -41.635128021240234, -14.165246963500977, 33.691123962402344, -18.63888931274414, 9.921222686767578, 31.31951904296875, -42.17354965209961, 13.039234161376953, -0.0752425342798233, -38.30626678466797, -26.052671432495117, -4.958448886871338, -27.739633560180664, 2.603971242904663, -0.44927436113357544, -20.81806755065918, -13.738263130187988, -13.805581092834473, -32.11368942260742, 43.43156051635742, 33.88287353515625, 24.766538619995117, 6.861879825592041, -37.19874954223633, 28.203096389770508, -72.41907501220703, -5.669344425201416, -18.78008270263672, 40.74635696411133, 6.566400051116943, 16.420570373535156, -1.231854796409607, 37.46846008300781, 27.994230270385742, 30.39818572998047, -3.2327728271484375, 20.876218795776367, 34.36341094970703, -3.5857396125793457, 14.055424690246582, -24.675687789916992, -26.57366943359375, -27.32088851928711, -13.118329048156738, -1.7560622692108154, 0.6898469924926758, 42.825706481933594, -5.184216022491455, 53.98295593261719, -27.82570457458496, 35.43600082397461, 5.743651866912842, -22.889387130737305, -7.5770063400268555, -39.57815170288086, 3.068479299545288, 13.290851593017578, -4.992623805999756, 1.590175986289978, 3.4188640117645264, -21.36227798461914, -28.123207092285156, 1.6886446475982666, 33.682552337646484, -19.256534576416016, 15.185855865478516, -13.705488204956055, -7.101999759674072, -76.96070098876953, 9.812299728393555, 12.278996467590332, 34.50407028198242, -29.705158233642578, -27.20684051513672, -3.3425261974334717, 16.6108455657959, -13.001081466674805, -26.194128036499023, 16.458967208862305, 12.761487007141113, 20.454212188720703, 14.164887428283691, 42.86091613769531, 10.1384916305542, 29.551538467407227, 2.4422857761383057, 22.4826602935791, 3.2318811416625977, 18.89296531677246, -15.727792739868164, 13.043571472167969, 27.567657470703125, -17.557676315307617, -34.816307067871094, 33.53025436401367, -8.849202156066895, 28.281492233276367, -11.674225807189941, 12.30579948425293, 16.369962692260742, -1.027243733406067, -42.894500732421875, -20.011390686035156, 1.5439858436584473, -21.665935516357422, -37.72388458251953, 15.665788650512695, -4.968860149383545, 21.687997817993164, -67.09850311279297, 7.024636268615723, -38.261749267578125, 12.554336547851562, -4.490817546844482, -15.594541549682617, -44.181697845458984, 50.36521911621094, -19.764240264892578, -2.631229877471924, 9.78637981414795, -49.42605972290039, -8.359850883483887, 25.503602981567383, 28.59219741821289, 21.06606101989746, 0.3457967936992645, 64.88839721679688, 28.776390075683594, 26.925607681274414, -15.837746620178223, -2.4252707958221436, 10.038863182067871, 65.37825775146484, -11.033327102661133, 25.83709716796875, -34.71870040893555, -42.342308044433594, 17.616802215576172, 30.078142166137695, -20.43728256225586, -8.554081916809082, 45.3655891418457, 20.778732299804688, -15.20988655090332, -76.25605773925781, 40.65684509277344, -26.599790573120117, 12.017682075500488, 11.31543254852295, -12.388321876525879, 31.150915145874023, 5.952480792999268, -18.531831741333008, -25.6977596282959, 25.702083587646484, -15.781251907348633, 10.781654357910156, -45.90245056152344, 17.912933349609375, -6.420698165893555, -8.272388458251953, 1.7427330017089844, -29.329933166503906, -18.369956970214844, 11.191003799438477, -18.773273468017578, 17.578340530395508, 24.547710418701172, -20.35661506652832, 3.8723788261413574, 18.962177276611328, -7.065470218658447, -21.76308822631836, 11.58374309539795, 10.335726737976074, 20.69818115234375, -29.5938777923584, -11.012956619262695, 2.463081121444702, 1.5436543226242065, 46.78952407836914, -26.50038719177246, -4.6286187171936035, 4.685633659362793, -2.951427459716797, 16.83355712890625, 12.51948356628418, 8.739545822143555, 3.529066562652588, -1.4700303077697754, 36.56966781616211, 16.25090789794922, 15.516752243041992, -1.5691282749176025, 15.139634132385254, -43.006160736083984, -4.8448662757873535, 6.632847785949707, 19.591754913330078, 12.564685821533203, -17.65361976623535, 36.670955657958984, -4.902684688568115, 11.124423027038574, 15.129273414611816, 3.316605806350708, -13.844701766967773, -56.68717575073242, 35.8771858215332, 101.68585205078125, 29.998659133911133, -32.71127700805664, 12.607736587524414, 0.9072563648223877, -19.0033016204834, 43.55570983886719, 34.49213409423828, -12.492403984069824, 17.02911376953125, 30.076278686523438, -30.485990524291992, 34.26879119873047, 18.886579513549805, -2.6628458499908447, -28.03667640686035, -3.8563268184661865, -47.34490203857422, 50.78254699707031, -29.41571617126465, -23.447877883911133, -11.626441955566406, 8.310565948486328, 81.54073333740234, -44.682003021240234, -44.61650085449219, 5.1503448486328125, -57.931884765625, 22.342243194580078, -2.7618346214294434, 15.037440299987793, 48.460750579833984, -21.32019805908203, -32.14274215698242, -11.486626625061035, -10.182703018188477, 13.111479759216309, -58.08792495727539, -8.878816604614258, 29.97197914123535, 2.9263339042663574, 1.117208480834961, -5.640257835388184, -9.095006942749023, -3.3663721084594727, -16.315237045288086, -37.90877151489258, 14.349897384643555, 28.516780853271484, 0.6470186114311218, -2.3328332901000977, 51.516395568847656, 15.069267272949219, 5.937427043914795, -41.9433708190918, 8.048192024230957, 53.494441986083984, 3.410083770751953, 8.167798042297363, -30.97515869140625, 47.615970611572266, 18.632076263427734, 40.838111877441406, -36.69999694824219, 58.548622131347656, 20.367719650268555, -47.104496002197266, 48.492618560791016, -2.488496780395508, 8.566932678222656, 48.148155212402344, -29.15663719177246, 8.927120208740234, 37.249366760253906, 13.744099617004395, 84.6378402709961, -75.3700942993164, -44.616172790527344, 1.8444843292236328, 30.283458709716797, 31.538307189941406, 40.992652893066406, -19.153465270996094, -11.383005142211914, 16.66182518005371, -19.554550170898438, -4.406308650970459, 20.976652145385742, 6.057043552398682, -7.274416446685791, 45.50579071044922, 32.891151428222656, 43.686920166015625, 37.990028381347656, -12.877421379089355, -24.58945655822754, -10.127740859985352, 10.528900146484375, -48.853126525878906, -76.41168212890625, -12.695500373840332, -5.316867828369141, -40.70899963378906, -5.579277038574219, -29.784015655517578, 39.917747497558594, 27.8153076171875, -47.98530578613281, 66.26750946044922, -28.42118263244629, 15.596564292907715, -61.392662048339844, 15.02173137664795, 3.6708641052246094, -43.43321228027344, -17.83714485168457, 30.83554458618164, -5.059340000152588, 17.121604919433594, 19.745655059814453, 58.81642532348633, -62.61223602294922, -25.574878692626953, 11.539737701416016, 9.82285213470459, -5.43319034576416, -14.209744453430176, -19.000581741333008, 4.767029285430908, 18.343900680541992, 10.724740982055664, -4.800041675567627, 2.6269032955169678, 32.71815872192383, 23.951887130737305, -31.383651733398438, 15.072433471679688, 4.315725803375244, 4.945849418640137, 10.109834671020508, -5.810227870941162, -28.778419494628906, -32.69309616088867, 6.724625587463379, 33.12094497680664, -46.97871398925781, 23.810043334960938, 46.37997055053711, -24.00737953186035, 26.287899017333984, -31.288862228393555, 47.000022888183594, -14.81501293182373, -16.24213409423828, -19.156293869018555, 0.5509399175643921, 79.60311889648438, -66.52748107910156, 8.685412406921387, -3.1469945907592773, -29.934581756591797, -26.063390731811523, 34.65993118286133, 14.589385032653809, -14.312132835388184, -17.025249481201172, 2.905571460723877, -39.80799865722656, -4.135910511016846, -46.298973083496094, -45.67264938354492, -13.41332721710205, -6.513472557067871, 6.261781692504883, 27.600162506103516, -55.408477783203125, -16.770776748657227, 38.474178314208984, 37.791664123535156, 30.51840591430664, -23.321088790893555, -2.986729145050049, 71.73218536376953, -44.03003692626953, 29.556743621826172, -29.75101089477539, -20.394775390625, -18.685726165771484, 5.701620101928711, -12.102476119995117, 17.02025032043457, -0.7422922253608704, -20.117895126342773, -46.03593063354492, 22.400178909301758, 0.04369974508881569, -37.243656158447266, -61.807472229003906, 28.972412109375, -6.700712203979492, 6.522860050201416, 6.674651622772217, 12.95210075378418, 44.082942962646484, -35.96171951293945, -13.557982444763184, -15.37847900390625, -23.111186981201172, -18.232990264892578, -3.7551493644714355, 24.610332489013672, 8.993860244750977, -25.09372329711914, -3.9003918170928955, -6.705164432525635, -21.26237678527832, 5.644212245941162, -2.1037683486938477, 41.1552734375, -50.795413970947266, 5.354482650756836, -38.572078704833984, -12.074705123901367, -5.1433281898498535, 5.208130836486816, 5.536305904388428, 16.605119705200195, -27.067340850830078, -46.52391052246094, 1.0822831392288208, -1.9293471574783325, 59.91455841064453, 54.63421630859375, -8.335274696350098, 14.567222595214844, -37.44789123535156, -51.58064651489258, -43.82952880859375, -26.708131790161133, -11.835179328918457, 5.195102214813232, -62.22968673706055, 41.7099723815918, 5.862701416015625, -15.454663276672363, 31.67012596130371, 37.68687057495117, -3.9863178730010986, 7.2262067794799805, 5.977774620056152, 50.46465301513672, -40.67951202392578, 19.057531356811523, -24.349231719970703, 21.052534103393555, 19.59331703186035, 38.997642517089844, 33.86614227294922, -8.36448860168457, 10.475147247314453, 17.8571834564209, 12.03996467590332, 38.67011642456055, 3.1655938625335693, -20.314254760742188, -66.99627685546875, -33.445045471191406, -55.58340072631836, 18.574539184570312, -32.99370574951172, 39.69929504394531, 23.78960609436035, -16.729167938232422, 0.6371427774429321, -32.284034729003906, -10.19025707244873, 8.678987503051758, -3.57088565826416, -38.96055221557617, 46.560401916503906, 26.589391708374023, 9.178060531616211, -8.934643745422363, -6.874910354614258, -1.603895664215088, -33.07050704956055, 15.547563552856445, 74.6010513305664, -18.80495262145996, 36.055625915527344, 11.922248840332031, -5.385247230529785, 20.557968139648438, -13.758193016052246, -53.89670944213867, -3.9658806324005127, -11.68930435180664, 0.5838878154754639, 42.60087585449219, -76.20008850097656, -15.327877044677734, -9.442116737365723, 38.75458908081055, -51.689884185791016, 47.85450744628906, -13.290963172912598, -38.12467575073242, -18.72719383239746, 16.70415496826172, -45.102909088134766, 1.061181902885437, 56.01676940917969, 7.9635186195373535, -30.388395309448242, -35.8248291015625, 8.078458786010742, -60.20188903808594, -4.535941123962402, -44.52781677246094, -28.864246368408203, 2.47163987159729, 4.124090194702148, -4.9202189445495605, 0.37588098645210266, -17.026432037353516, 30.275238037109375, 22.043357849121094, -49.99314498901367, 36.52437210083008, 7.537250518798828, -6.845951080322266, -0.9910691976547241, 53.59672546386719, -2.8513731956481934, -1.5414432287216187, -50.71083450317383, 33.728641510009766, -54.207252502441406, 14.941267013549805, -33.5670051574707, -2.8207998275756836, -14.380119323730469, 1.5823101997375488, 37.16509246826172, 9.22152328491211, -2.098813056945801, -0.09100162237882614, -1.4159718751907349, -26.21649169921875, -9.622328758239746, 14.271440505981445, -32.443214416503906, 9.669502258300781, 50.5947151184082, 47.28813934326172, -25.44785499572754, 15.720673561096191, 14.198837280273438, 38.48374557495117, -5.087408542633057, 112.31636047363281, -5.179886817932129, -0.4377325773239136, -14.21307373046875, 17.602684020996094, 22.1365909576416, 1.5289949178695679, 30.362051010131836, 15.50540542602539, -28.571849822998047, 17.74717903137207, -10.411894798278809, -23.73443603515625, 71.13507080078125, 15.594181060791016, 11.672436714172363, 43.7735710144043, -26.43693733215332, 21.525035858154297, 33.120967864990234, -0.0007776026614010334, 36.339500427246094, 32.69342803955078, 41.87925720214844, -40.98336410522461, 9.282198905944824, 28.731081008911133, -51.328277587890625, 24.21567153930664, -0.26432740688323975, -31.488479614257812, -46.15044021606445, -21.767423629760742, 45.159034729003906, 20.99677085876465, 14.00404167175293, -29.29155731201172, -22.899097442626953, -10.366975784301758, -9.05477237701416, -34.15618133544922, -12.956449508666992, 58.1723518371582, -1.8479111194610596, -4.526672840118408, 27.164348602294922, 6.100858211517334, -38.02906799316406, -8.487903594970703, -5.584955215454102, 26.33942413330078, 23.315479278564453, 24.06253433227539, 19.768524169921875, -36.14765167236328, 23.347976684570312, 21.45365333557129, -25.159221649169922, 30.881906509399414, 19.434268951416016, -48.47051239013672, -1.2441636323928833, 8.074233055114746, 56.30959701538086, 13.162172317504883, -38.606788635253906, 13.764833450317383, -23.543067932128906, 24.34756851196289, 47.71836471557617, -11.951425552368164, -9.425146102905273, 6.545623302459717, -14.017298698425293, 15.848225593566895, -13.500130653381348, 51.957759857177734, 21.36880111694336, -20.299087524414062, 10.093384742736816, -22.610069274902344, -39.557273864746094, -22.37763023376465, 19.693721771240234, -37.94731521606445, 15.138553619384766, 46.71773147583008, 1.0720641613006592, -40.94987106323242, 6.667194843292236, -50.25906753540039, 6.444462776184082, 12.752016067504883, 31.57135581970215, 48.61391067504883, 20.005014419555664, -28.92742919921875, 7.689387321472168, 42.97526168823242, 29.330120086669922, -15.468557357788086, 25.129056930541992, 21.276018142700195, 44.36769104003906, 25.56114959716797, 17.96660804748535, -33.18120574951172, -55.329105377197266, -15.922469139099121, 2.5840017795562744, 1.4282125234603882, -3.7018024921417236, 50.15849685668945, -39.745967864990234, -7.522859573364258, -13.308647155761719, 96.63237762451172, 11.139419555664062, -9.61035442352295, 9.522443771362305, -17.00216293334961, -17.479862213134766, 24.676050186157227, 21.534181594848633, -57.974029541015625, -60.8228874206543, 43.24319076538086, -22.197460174560547, 4.1473307609558105, -12.303339004516602, 32.953636169433594, 2.672333240509033, 28.872941970825195, -10.471992492675781, 2.459432601928711, -13.14622974395752, 10.509186744689941, -29.20005226135254, 23.68107795715332, 17.96225929260254, 10.476411819458008, 39.529808044433594, 8.07758617401123, -42.02106475830078, -12.447996139526367, -18.29867935180664, 0.4798586964607239, -15.008813858032227, -8.057950973510742, -22.343122482299805, -53.398563385009766, -0.7045078873634338, -21.189233779907227, -1.9364713430404663, -23.766555786132812, -20.57550811767578, -50.14622497558594, -39.25575256347656, 0.8661056756973267, 12.484792709350586, -12.838406562805176, 18.59098243713379, 6.751656532287598, -37.579593658447266, 15.706329345703125, 4.3713812828063965, -6.3306050300598145, 42.20334243774414, -42.3871955871582, -36.36620330810547, 25.007463455200195, -5.174779415130615, 4.2300262451171875, -10.784578323364258, 18.20386505126953, -12.230148315429688, 36.02701950073242, 2.4693644046783447, -23.854673385620117, -10.79222583770752, -29.90165138244629, 21.46283721923828, -2.10931658744812, -48.64167404174805, -55.483551025390625, 21.38471794128418, 8.243249893188477, 4.587670803070068, -0.13309478759765625, -21.801645278930664, -67.95021057128906, -9.707984924316406, 33.97990798950195, -34.29173278808594, 1.6393040418624878, -42.579490661621094, 14.87540054321289, 16.74720001220703, 11.489165306091309, 5.496997356414795, -3.755580425262451, 40.121299743652344, -82.9201431274414, -32.08842849731445, -29.184913635253906, -29.106029510498047, 29.805809020996094, 29.968950271606445, -48.40189743041992, -9.595755577087402, -6.567149639129639, 35.834800720214844, -27.54509735107422, -47.3702507019043, 40.318809509277344, -5.905087471008301, -33.550636291503906, 18.15163803100586, -39.75941467285156, -46.96595764160156, 30.69231414794922, 12.786741256713867, 26.48147964477539, 1.7825324535369873, 24.502330780029297, 32.19918441772461, -2.892888307571411, -15.192984580993652, 32.57700729370117, -6.40459680557251, 15.555418968200684, 32.6297721862793, 32.536346435546875, -7.907729625701904, -29.082460403442383, -22.92882537841797, 19.185081481933594, -35.54109573364258, 27.900087356567383, 54.15049743652344, -14.89505386352539, -40.38542938232422, -58.76338195800781, 31.413753509521484, -45.53044891357422, 10.008763313293457, 14.17584228515625, 37.85679626464844, 33.5792236328125, -17.33283042907715, 20.663005828857422, 42.38206481933594, -33.86686325073242, -3.113961696624756, -22.998626708984375, 2.418410062789917, 20.942941665649414, -59.07522201538086, -5.408432483673096, 6.171579360961914, -21.492595672607422, 13.557731628417969, -6.740297794342041, -14.402061462402344, 13.004915237426758, -22.490528106689453, 20.496498107910156, -4.400664329528809, -12.745280265808105, -23.351398468017578, 33.48043441772461, 8.706477165222168, 14.975192070007324, -34.776329040527344, -6.419101715087891, 25.86651611328125, 3.851161241531372, 21.530048370361328, -40.98837661743164, 43.179866790771484, -10.887409210205078, -9.303699493408203, 6.835683822631836, 43.11857986450195, -19.525175094604492, 8.017769813537598, -32.39158248901367, 41.903106689453125, -5.676278591156006, -24.591575622558594, -29.029129028320312, 24.811824798583984, -36.81181335449219, 15.23193073272705, 40.812217712402344, -62.98630142211914, -4.625205039978027, 17.203304290771484, 14.224705696105957, -9.395634651184082, -4.339093208312988, 61.05144119262695, -11.092225074768066, -5.420440196990967, 39.06763458251953, -21.95254135131836, -13.840291976928711, 14.000784873962402, -6.700634002685547, -34.96441650390625, 20.229257583618164, 17.1333065032959, 21.62073516845703, -17.120925903320312, 6.627711296081543, 22.180444717407227, -29.329654693603516, 3.194096565246582, 12.38819694519043, 17.895185470581055, -1.2865536212921143, -95.24075317382812, -72.99935150146484, -61.234474182128906, 10.561381340026855, 11.714913368225098, 41.422096252441406, -27.454315185546875, 27.499088287353516, 18.544879913330078, 10.744241714477539, -23.553142547607422, 5.413240909576416, -19.46201515197754, 22.309295654296875, -82.36351776123047, 15.8026762008667, -15.110882759094238, 45.00475311279297, 4.625855445861816, 5.607691287994385, 20.252103805541992, 37.19526672363281, 21.604787826538086, -36.41937255859375, -31.08808708190918, 19.261425018310547, 5.254418849945068, -14.423507690429688, 35.85243225097656, 27.963729858398438, -21.776201248168945, -18.514867782592773, 32.566062927246094, 26.021217346191406, -14.069502830505371, -0.929615318775177, -0.4062241017818451, 11.938370704650879, -36.21842575073242, 3.354058265686035, -19.827604293823242, 50.26066207885742, -32.36314392089844, 53.54045486450195, -55.18101119995117, 47.11918640136719, -25.422367095947266, -6.71354866027832, -17.717565536499023, 72.98907470703125, -14.436059951782227, 43.93321228027344, 12.138627052307129, 10.234886169433594, -28.504281997680664, 10.450400352478027, 1.4880436658859253, -25.15447235107422, -13.68436050415039, -40.57957458496094, -33.78031921386719, 9.70142650604248, 31.396154403686523, -15.018206596374512, 31.63630485534668, 0.3626331686973572, 43.22233200073242, 35.545509338378906, 35.03255081176758, 4.280186176300049, -35.829776763916016, 22.23526382446289, 21.43630599975586, -12.156774520874023, 38.934295654296875, -50.948944091796875, 12.876999855041504, 11.990044593811035, -49.78439712524414, -41.80971145629883, -28.650217056274414, -28.8658447265625, 48.04273986816406, 22.531267166137695, -5.229418754577637, 25.57892608642578, 14.142845153808594, -39.550697326660156, -48.32548141479492, 4.701493263244629, 24.555604934692383, 3.1244418621063232, 48.09435272216797, 35.74809646606445, -19.67321014404297, 35.03697204589844, -12.857579231262207, -20.64820671081543, 23.57086944580078, 11.233389854431152, 27.868921279907227, -1.3150628805160522, -40.597923278808594, 28.668472290039062, 41.14884948730469, -1.3639167547225952, 20.53455352783203, -66.13207244873047, -23.007854461669922, 22.537982940673828, 14.466169357299805, -4.52346658706665, 47.171592712402344, -13.771608352661133, -33.5906867980957, -17.867746353149414, -29.972148895263672, 14.002527236938477, 24.525861740112305, 26.46666717529297, -26.44083595275879, 7.353426933288574, -27.41204833984375, 18.88227081298828, 15.04709243774414, -25.953269958496094, -16.452131271362305, -16.11788558959961, -4.3689045906066895, -37.02079772949219, 39.113502502441406, -18.000404357910156, 21.563772201538086, -6.839354515075684, 8.141470909118652, -7.992812156677246, -52.835601806640625, 15.597431182861328, 35.410865783691406, 94.59635162353516, -19.426605224609375, 23.908180236816406, 3.5263559818267822, 16.828901290893555, -11.270048141479492, 26.17344856262207, 48.3836669921875, 34.80055236816406, 1.977874994277954, 1.3709195852279663, 38.81379699707031, -13.466179847717285, -69.11617279052734, -62.81587600708008, -27.978744506835938, 42.440765380859375, 35.141109466552734, 9.768898010253906, -21.693151473999023, 43.12558364868164, -4.656032562255859, -19.874488830566406, -51.9529914855957, 19.82634735107422, -25.87872886657715, -0.8349412679672241, -53.07715606689453, -38.09882736206055, -9.852330207824707, 4.647870063781738, 10.82114028930664, 43.9088020324707, 44.21918869018555, 12.153623580932617, 1.8113784790039062, -32.33811569213867, -27.017627716064453, 12.894171714782715, -12.206820487976074, -20.512344360351562, 19.825586318969727, -18.640811920166016, -35.3000602722168, 3.8244214057922363, 38.3328857421875, -32.48735046386719, -7.7608866691589355, 16.785606384277344, -24.78627586364746, 11.791332244873047, 66.43425750732422, 25.936967849731445, -57.88380432128906, -38.34666442871094, 45.799957275390625, 85.16000366210938, 23.37424659729004, 11.294112205505371, 43.17025375366211, -12.355220794677734, -17.536762237548828, -33.58022689819336, -45.75433349609375, -27.364736557006836, 16.081396102905273, 2.701107978820801, 22.433019638061523, 22.08443832397461, 59.213340759277344, 33.310604095458984, 14.644776344299316, 16.13585662841797, 29.04439926147461, -31.00354766845703, -1.5535768270492554, 34.85477066040039, -6.122291088104248, -21.64688491821289, -67.78031921386719, -5.035433292388916, -37.132930755615234, 2.8421425819396973, 31.071195602416992, 4.014398097991943, 13.411848068237305, -24.575292587280273, 23.691722869873047, -40.167816162109375, 2.346531867980957, 40.09034729003906, -14.25762939453125, 17.481958389282227, -0.9805978536605835, -1.3748092651367188, -15.893084526062012, -13.160933494567871, 48.71868133544922, 31.138187408447266, -0.354181170463562, -13.925755500793457, 57.165748596191406, -42.44601821899414, -7.338042736053467, -3.54388165473938, -26.068212509155273, 1.6179282665252686, 41.344329833984375, -67.05286407470703, -18.239221572875977, 2.1426355838775635, -32.706607818603516, 4.324816703796387, 8.906609535217285, -12.269038200378418, -20.813682556152344, -13.586273193359375, 1.410386323928833, -6.751243591308594, -17.574262619018555, 23.387168884277344, -55.491111755371094, 70.94621276855469, -44.555877685546875, 43.60316848754883, 44.49190902709961, 5.921711444854736, -30.511043548583984, 64.61124420166016, -41.3596076965332, -23.164287567138672, -13.542327880859375, 19.027860641479492, 70.29911804199219, -3.3582522869110107, 59.252891540527344, 23.006763458251953, 47.875213623046875, 26.68134117126465, 68.35554504394531, -24.00232696533203, -5.461647033691406, 17.246339797973633, 17.77179527282715, 38.76429748535156, -13.802947044372559, 1.8856329917907715, -19.49368667602539, -34.547908782958984, 17.404903411865234, -26.93671989440918, -12.559259414672852, -41.40736389160156, -4.805178165435791, -14.538226127624512, 19.408601760864258, 67.26470184326172, -57.78963851928711, -45.14786911010742, -8.214215278625488, -42.4655647277832, -22.781246185302734, 37.08974838256836, -48.36404800415039, -5.7330098152160645, -13.16296672821045, 13.920655250549316, -29.36798858642578, 30.13412094116211, -4.238818168640137, 12.946046829223633, 8.215167999267578, 0.2816488444805145, -21.03670883178711, 34.03970718383789, 11.37441635131836, -4.941600322723389, 20.668561935424805, 0.7702769041061401, 22.823884963989258, -43.945640563964844, -0.8231971859931946, -10.159401893615723, 30.050085067749023, 13.926656723022461, 29.17980194091797, 19.741127014160156, -35.2567138671875, 2.914346933364868, -20.387550354003906, -1.5045030117034912, 12.781731605529785, -27.451416015625, -24.85533905029297, 6.654277801513672, 26.899702072143555, -21.95686912536621, -3.660719394683838, 28.748071670532227, -39.087162017822266, -24.29025650024414, -37.63991928100586, -29.478763580322266, 42.671875, 67.11920928955078, -21.853900909423828, -14.050894737243652, -16.408485412597656, -38.74891662597656, -48.77153778076172, -13.149781227111816, -35.25196075439453, 24.169740676879883, -32.59595489501953, 21.04913330078125, 23.495027542114258, -36.453609466552734, 33.63202667236328, 8.322543144226074, 36.6636962890625, -8.672840118408203, -17.702709197998047, -24.735389709472656, -23.586183547973633, 15.818005561828613, 69.47506713867188, -5.338238716125488, 41.411781311035156, -22.05773162841797, 19.930517196655273, -29.063554763793945, -27.16619873046875, -74.46002197265625, -0.7962145805358887, 27.53538703918457, -4.896088123321533, -18.3084716796875, -5.0618672370910645, -20.65808868408203, -34.77155685424805, 0.37678781151771545, -28.619096755981445, -17.93962860107422, -61.06507873535156, 9.27375602722168, -0.3609849512577057, 18.36040496826172, -3.9645020961761475, -42.03166961669922, -33.818363189697266, -20.633670806884766, -9.084813117980957, -4.0308146476745605, 27.680654525756836, -11.388121604919434, 8.038330078125, 29.778602600097656, -29.12164878845215, 2.086468458175659, 10.30068588256836, -31.670183181762695, 50.869598388671875, 33.75947952270508, -13.995104789733887, 18.086828231811523, -67.25708770751953, 25.204431533813477, 25.85514259338379, 45.95482635498047, -4.583942890167236, -7.120738506317139, 32.315921783447266, -36.51937484741211, 4.636128902435303, 7.616482734680176, -2.7791237831115723, 9.071044921875, -2.4618608951568604, 49.69977569580078, -20.967906951904297, -34.52169418334961, -23.132184982299805, -38.36429977416992, 41.47037124633789, -20.60201644897461, -12.905177116394043, -1.522199273109436, 5.930512428283691, 59.3653678894043, -20.583507537841797, 2.6775736808776855, 17.424301147460938, 4.4823174476623535, 40.553497314453125, 37.95280075073242, -21.071107864379883, -37.642520904541016, -35.012657165527344, -18.529830932617188, -30.61151885986328, 6.601629257202148, -17.659976959228516, 8.639986038208008, 5.93756103515625, -13.86185073852539, 47.80094909667969, 34.719337463378906, 17.606143951416016, 9.292567253112793, -32.5477409362793, 28.91281509399414, -67.69525146484375, 21.060283660888672, -36.77942657470703, -11.247450828552246, -20.673633575439453, 37.98076248168945, 59.095252990722656, 60.75432205200195, -23.506389617919922, 29.000524520874023, -21.68655776977539, 28.339649200439453, 35.50174331665039, 45.42142868041992, -34.54515838623047, 30.35516357421875, 0.11378639936447144, -0.30413177609443665, -9.288735389709473, 11.190422058105469, -33.77781677246094, 0.5085567235946655, -10.94665241241455, 18.861316680908203, -4.193841934204102, 16.028202056884766, -19.893482208251953, -31.398256301879883, -6.542622089385986, 31.528972625732422, 28.579801559448242, -59.86326217651367, 9.315210342407227, 7.710249423980713, 34.54351806640625, 7.0406718254089355, -13.418535232543945, 11.784282684326172, 15.46086597442627, -20.045663833618164, -1.1029226779937744, -34.55474853515625, 22.928178787231445, 1.2475337982177734, 20.739086151123047, -45.497840881347656, 22.947105407714844, -1.2774226665496826, 20.003921508789062, -12.808046340942383, 43.13914108276367, 14.080802917480469, -7.366737365722656, -5.092803478240967, 4.013007164001465, 54.19451141357422, 46.01337432861328, 3.0394811630249023, 17.344207763671875, -20.47161102294922, 5.562313556671143, 53.26239776611328, -12.746451377868652, -20.633304595947266, 76.10256958007812, 2.688926935195923, 63.26998519897461, 27.744211196899414, -5.0430498123168945, -58.63251876831055, 37.783443450927734, -12.110660552978516, -44.92288589477539, 4.844037055969238, 19.247596740722656, -36.244903564453125, -28.728626251220703, 11.163399696350098, 40.020416259765625, 41.98707962036133, 10.51033878326416, 8.319788932800293, 36.03276062011719, 9.151167869567871, -4.550346374511719, 74.56925964355469, 4.011045932769775, -25.96658706665039, -3.0239014625549316, 2.195563554763794, 13.12105941772461, 13.958443641662598, -21.73451042175293, -19.286182403564453, -56.78560256958008, -26.554018020629883, -28.269012451171875, -24.840856552124023, 47.90354919433594, -46.08841323852539, 11.096514701843262, -19.259321212768555, -0.3134283423423767, -13.44847583770752, 42.03889846801758, -17.13405990600586, 4.018756866455078, -12.30580997467041, 71.9975357055664, 34.08815383911133, 20.711837768554688, -8.641056060791016, -23.50843048095703, -8.100065231323242, -5.164677619934082, 7.926363945007324, 18.23414421081543, -27.415409088134766, -4.6647868156433105, 7.118567943572998, 13.206162452697754, 23.279499053955078, -20.181320190429688, -60.667396545410156, 14.11917495727539, 1.3324521780014038, 20.156606674194336, -21.2379150390625, -7.870980739593506, -54.86079788208008, -19.321565628051758, -46.487422943115234, -8.894638061523438, -44.55153274536133, 36.60733413696289, 3.8641629219055176, 39.82429122924805, 45.191017150878906, -15.073408126831055, 4.740246772766113, 16.004230499267578, -8.086459159851074, -9.895604133605957, -25.909954071044922, -44.18672180175781, -13.920950889587402, -20.875953674316406, 28.66720962524414, 55.873130798339844, 5.868587017059326, -48.75788497924805, -35.33523941040039, -11.388240814208984, 8.768168449401855, 30.398765563964844, 39.77010726928711, 12.54775333404541, -15.471149444580078, 23.89017677307129, -4.076815605163574, -43.53449249267578, 71.45431518554688, -2.697589874267578, -8.97564697265625, -17.925640106201172, 22.166032791137695, -7.8458709716796875, 40.30356216430664, -16.010175704956055, -30.776853561401367, 42.79769515991211, 14.19698429107666, -42.976348876953125, 23.662832260131836, 35.2298583984375, -24.990602493286133, 54.510772705078125, -2.0973832607269287, -8.215560913085938, 27.91854476928711, 8.721961975097656, -5.179366588592529, 17.53493881225586, -65.71090698242188, 47.418697357177734, -54.14452362060547, 5.9537434577941895, -12.316536903381348, 21.842172622680664, -11.601595878601074, 79.79681396484375, 36.15851593017578, 58.39173889160156, 17.59754180908203, -56.558677673339844, 9.711102485656738, -10.288590431213379, 18.84600067138672, 19.68421745300293, -3.7772445678710938, -25.933326721191406, 1.3041855096817017, 11.443625450134277, -16.45852279663086, 19.01441764831543, -19.0324649810791, 9.271732330322266, 8.232044219970703, 6.401240348815918, 43.88972473144531, 32.846458435058594, 3.8967673778533936, -12.602737426757812, 36.59103775024414, 10.288641929626465, -28.80835723876953, -21.432039260864258, 33.46467590332031, 36.25090026855469, 7.763158798217773, 6.572495460510254, 37.569515228271484, 54.26888656616211, -42.39255905151367, 38.69721603393555, -26.85742950439453, -7.524011135101318, 19.4547119140625, -28.394351959228516, 28.123756408691406, 44.285831451416016, 28.788776397705078, -34.66441345214844, 0.9320051670074463, -3.4579083919525146, 38.005802154541016, -12.212430000305176, 91.01913452148438, 14.327115058898926, 2.8243064880371094, 5.960423946380615, -0.7440109848976135, -15.721713066101074, 12.723800659179688, -11.688422203063965, 31.11111831665039, -75.10316467285156, -0.64145427942276, 25.08283042907715, 44.46029281616211, 4.863824367523193, -19.257766723632812, -3.2797677516937256, -49.238773345947266, 61.65357971191406, 12.983882904052734, 57.94290542602539, -22.01459503173828, 19.73750114440918, -20.76102066040039, -0.6438709497451782, -27.320722579956055, -4.545160293579102, 49.787315368652344, -0.30866238474845886, -23.583681106567383, 5.165079116821289, 12.991254806518555, 22.01258659362793, 29.242530822753906, 27.441801071166992, 54.833518981933594, -11.06691837310791, 12.936549186706543, -16.80876350402832, -15.270425796508789, -48.51373291015625, 8.159513473510742, 2.4563276767730713, 50.61757278442383, -4.135773181915283, -3.103992462158203, -13.964279174804688, -43.177860260009766, 48.740474700927734, -7.12383508682251, 1.871254563331604, 14.925207138061523, 2.9112348556518555, 13.498549461364746, 38.75164794921875, 15.736531257629395, 11.70501708984375, -20.61176300048828, -44.39857482910156, -26.868453979492188, -16.16743278503418, -25.610668182373047, -26.545907974243164, 50.89684295654297, -34.48759078979492, -15.165792465209961, 20.747669219970703, -14.712491989135742, 24.840072631835938, -57.54552459716797, 7.699218273162842, -26.435768127441406, 36.79530334472656, -28.17639923095703, -47.69141387939453, 5.100710868835449, -12.705543518066406, 17.63648796081543, -15.505810737609863, 76.23648071289062, 9.631129264831543, -10.387063026428223, 5.249756336212158, 22.865028381347656, 8.705644607543945, 27.197996139526367, 11.119443893432617, -52.544883728027344, 47.06220245361328, 3.120516061782837, 14.063173294067383, 12.140623092651367, -25.77029037475586, 0.8878577351570129, 21.606632232666016, 59.025840759277344, 26.866981506347656, 1.9276949167251587, 31.40943717956543, 42.5157470703125, 9.684160232543945, -9.369669914245605, 82.61598205566406, -7.53547477722168, -19.36247444152832, 10.0346040725708, -8.04875659942627, 18.141847610473633, -44.54358673095703, 40.759342193603516, -43.26873016357422, -24.55573081970215, 11.446015357971191, 2.660752534866333, 15.172637939453125, 8.422067642211914, -1.9794049263000488, -47.87717819213867, -30.143291473388672, -33.073604583740234, 17.767580032348633, -28.55129623413086, -31.876630783081055, -82.94202423095703, -16.07309341430664, 12.20392894744873, 28.520299911499023, -11.230243682861328, 18.35875701904297, -1.3663146495819092, 12.721041679382324, -40.53371047973633, -18.538490295410156, -16.152088165283203, -36.00819778442383, -14.845921516418457, 6.371760368347168, 2.9899790287017822, -2.722989320755005, 51.32343673706055, -35.54408264160156, 23.839555740356445, 6.568514347076416, -23.08951759338379, 18.43252944946289, 60.98515701293945, -59.146148681640625, 75.66897583007812, -12.231620788574219, -11.766454696655273, -19.79041290283203, -22.0198917388916, -60.7232666015625, -5.196331977844238, 45.60130310058594, -14.02728271484375, -18.29098129272461, -18.544578552246094, 10.146003723144531, -46.07656478881836, 20.066627502441406, -20.947940826416016, 28.84095573425293, 1.322850227355957, -58.862709045410156, 52.07204055786133, -9.575393676757812, -4.248874664306641, 31.21295166015625, 29.94556427001953, 41.341121673583984, 3.4433228969573975, 27.840744018554688, -10.75220775604248, 4.870303630828857, 13.154860496520996, 28.604089736938477, -46.9399528503418, 34.273414611816406, -33.686954498291016, -10.44929027557373, -10.84394645690918, 48.57567596435547, 53.29352951049805, 24.617822647094727, 5.746391773223877, 15.108193397521973, -26.208112716674805, -37.9043083190918, -23.577333450317383, 53.41218566894531, 6.865702152252197, 0.34356310963630676, -0.7567060589790344, 34.934783935546875, 23.212379455566406, 10.897997856140137, -45.783084869384766, 12.944181442260742, 5.407020568847656, -18.056190490722656, 44.46229553222656, -37.338966369628906, 19.025432586669922, -33.98389434814453, 10.606578826904297, -10.034139633178711, -51.35801315307617, 10.305679321289062, 11.701386451721191, 33.58879089355469, 8.019325256347656, 24.539199829101562, 21.037517547607422, -14.213424682617188, 44.08714294433594, 33.27768325805664, 43.682430267333984, 5.778316974639893, 53.324771881103516, -59.96513366699219, -25.651599884033203, -9.86072063446045, 56.844085693359375, 25.29846954345703, 48.45582580566406, -25.385162353515625, 3.056058645248413, -40.22870635986328, -3.136383295059204, -18.65371322631836, 38.222896575927734, 5.3555707931518555, 3.0468695163726807, 63.93549346923828, -21.63078498840332, -30.172731399536133, -9.29566478729248, 4.2371416091918945, 36.8306770324707, 85.89894104003906, -2.1126606464385986, -57.570064544677734, 29.809532165527344, -25.200267791748047, 33.350616455078125, -18.405996322631836, -28.40593910217285, 18.202909469604492, -35.86957931518555, -15.052478790283203, -18.35955238342285], [-24.56892967224121, 19.457229614257812, 7.786358833312988, 38.596954345703125, 62.898529052734375, -7.652842998504639, -31.00047492980957, 4.283366680145264, -18.725706100463867, 0.19326668977737427, -0.22172732651233673, 33.513710021972656, 11.289860725402832, -42.63945770263672, -9.650420188903809, 3.597776174545288, 26.91878890991211, -64.44719696044922, -26.820470809936523, -21.72355842590332, 2.2259275913238525, -8.685604095458984, -10.3010835647583, 12.464099884033203, 67.0447006225586, -23.113590240478516, -88.87257385253906, 17.874435424804688, -17.742338180541992, -44.88939666748047, -11.712934494018555, 9.436406135559082, -14.100118637084961, -13.41861629486084, -15.82175350189209, -24.897397994995117, 94.9594497680664, 22.71625518798828, 38.80891418457031, 48.63064956665039, 12.529036521911621, 13.363404273986816, -55.93851089477539, 8.80483341217041, 4.878212928771973, 46.377723693847656, -75.61641693115234, 17.59356117248535, -12.69219970703125, -50.11891174316406, -1.1160037517547607, 22.15968894958496, -18.49217987060547, -17.95074462890625, -27.295604705810547, 45.58109664916992, 9.719338417053223, -13.895115852355957, 18.990711212158203, 52.206146240234375, 31.429996490478516, -11.291808128356934, 0.5314465165138245, 43.927001953125, 2.344726085662842, 31.730796813964844, -18.71373748779297, 10.993313789367676, 5.4413042068481445, 12.707558631896973, 28.41837501525879, -26.553956985473633, 69.6329116821289, 13.73449420928955, -44.29798889160156, -6.510408401489258, -21.3101863861084, -9.368400573730469, 6.479245662689209, 5.21280574798584, 6.45580530166626, 21.181406021118164, -52.79379653930664, 33.481746673583984, -17.856718063354492, 15.88713550567627, 5.918125629425049, 29.8045597076416, -28.501304626464844, -23.65854263305664, -0.2712787985801697, -4.808859825134277, -48.296913146972656, -47.18616485595703, -15.325992584228516, 18.54332160949707, 9.082374572753906, 12.137809753417969, 26.611103057861328, -20.740461349487305, 23.953155517578125, 47.1829833984375, -16.4630126953125, -11.312163352966309, 3.0300450325012207, -5.2345147132873535, -59.6624870300293, -5.5361552238464355, -33.73118591308594, 18.068851470947266, -1.9179250001907349, 6.934837341308594, -18.957839965820312, -0.8094019889831543, -32.94484329223633, -18.801603317260742, 0.9750122427940369, 48.26268005371094, 4.975608825683594, -38.07963180541992, 8.482614517211914, 4.384841442108154, -25.709619522094727, -0.33467555046081543, -9.2902250289917, 43.19733810424805, 5.179398059844971, 41.91709518432617, -17.365642547607422, -12.197210311889648, -12.35484504699707, 42.00825500488281, 28.662899017333984, 11.267109870910645, 10.590869903564453, -25.583881378173828, -12.252182960510254, -7.678002834320068, -7.049763202667236, -3.0068185329437256, 39.95595932006836, 2.4015719890594482, 26.117094039916992, -29.50820541381836, -18.872020721435547, -6.927237510681152, 57.2267951965332, 28.887399673461914, 26.07928466796875, -27.725963592529297, 13.712879180908203, -22.166934967041016, 16.84266471862793, 27.03688621520996, 58.048404693603516, -7.805339813232422, 22.280853271484375, -18.586122512817383, 11.792675971984863, -19.91103172302246, -1.434148907661438, -8.99001407623291, -0.7175977230072021, -27.367061614990234, -29.46596336364746, -28.716054916381836, -30.95333480834961, 36.6579475402832, 51.025474548339844, -66.4208755493164, -12.33106803894043, -25.677650451660156, 22.187788009643555, -41.942630767822266, 63.642574310302734, 38.09729766845703, -35.04647445678711, -40.46173095703125, 9.089344024658203, 16.26593780517578, -55.5592041015625, -3.391829252243042, -4.377121448516846, -28.122774124145508, 0.2874092161655426, -26.176464080810547, 1.0381993055343628, -0.6174584627151489, 1.1295808553695679, 4.2936482429504395, -30.74816131591797, 5.826531887054443, 26.260746002197266, -1.9974336624145508, -0.8906182050704956, 8.734686851501465, -20.68008804321289, 23.297096252441406, -51.54179763793945, 0.3339279294013977, 16.93373680114746, 2.0958988666534424, -30.427536010742188, -45.65159606933594, 1.1820461750030518, -55.73351287841797, -1.7755533456802368, -7.159961700439453, -9.767402648925781, -13.9929838180542, 14.968149185180664, 5.1628875732421875, -0.6296122074127197, 37.88642883300781, -9.948091506958008, -46.12158966064453, -6.965053558349609, -11.862338066101074, -10.108902931213379, -14.280736923217773, -7.677427291870117, -8.38258171081543, 15.315329551696777, -21.21486473083496, -7.895573139190674, 25.710111618041992, -25.166860580444336, 8.096806526184082, -0.6378360390663147, 13.655708312988281, -20.337005615234375, 34.179412841796875, 18.096607208251953, -74.53321838378906, -29.819822311401367, 14.364845275878906, -10.32470417022705, -24.920120239257812, -34.70652770996094, 34.146934509277344, -38.68480682373047, 16.549894332885742, -16.68600082397461, -41.97465896606445, 10.409374237060547, -10.590991973876953, 23.822341918945312, -39.76280975341797, -33.967994689941406, -21.884672164916992, 28.715120315551758, -28.259605407714844, 77.434326171875, 24.403879165649414, -16.316007614135742, -44.2658576965332, 29.949764251708984, 17.384204864501953, 36.233516693115234, -5.745515823364258, 20.188709259033203, -15.16272258758545, -4.718964099884033, -18.710744857788086, -85.67096710205078, 17.198827743530273, 35.32069778442383, 49.374176025390625, 19.592496871948242, 1.3469456434249878, -36.34481430053711, 58.65110397338867, -11.493515968322754, 10.05475902557373, 14.032629013061523, -28.956884384155273, -19.76104736328125, 18.254186630249023, 25.63191795349121, 27.395944595336914, 4.131923198699951, -23.643091201782227, 72.50791931152344, 27.086946487426758, 15.836579322814941, -4.487335205078125, -13.519754409790039, 10.059736251831055, -35.73923873901367, 39.77534484863281, -38.09889221191406, 36.73859405517578, -11.77822494506836, 32.622230529785156, 33.120723724365234, 13.352099418640137, 60.632205963134766, 47.84215545654297, 18.53230857849121, -27.0028133392334, 3.590276002883911, 14.915738105773926, 32.788047790527344, -6.283078193664551, 34.25273895263672, 19.966171264648438, 29.00029945373535, 11.595111846923828, 57.50600051879883, 3.902894973754883, -31.986053466796875, -2.2044456005096436, 60.06084060668945, -44.55888366699219, -56.355987548828125, -5.359774589538574, -17.420080184936523, -23.64020347595215, 25.65083885192871, 7.568790435791016, 14.401172637939453, 6.530178070068359, -4.785772800445557, 0.14900588989257812, 17.266698837280273, -46.99483108520508, 9.777104377746582, 43.74512481689453, -32.18992614746094, 14.748064041137695, -9.732451438903809, 32.90666580200195, -18.749662399291992, -16.057470321655273, 4.402360439300537, -12.538643836975098, 1.9492480754852295, 88.42189025878906, -0.8496558666229248, 24.445043563842773, -41.07329559326172, -14.189362525939941, 27.27840805053711, -6.142236232757568, 4.050001621246338, 25.42798614501953, -4.803072929382324, 31.17901611328125, 3.7326419353485107, 24.266265869140625, 24.354238510131836, 6.279508590698242, -24.107553482055664, 9.214203834533691, -22.74016761779785, 33.59110641479492, 9.470260620117188, -62.996063232421875, 34.372535705566406, -53.20188522338867, -1.7288235425949097, 45.345367431640625, -14.249600410461426, -38.595943450927734, 22.860946655273438, -17.17704200744629, -10.658923149108887, 8.40653133392334, -3.745105743408203, 1.2441391944885254, -34.446128845214844, 1.7969919443130493, 5.3003315925598145, -15.806302070617676, 1.1041244268417358, -21.539148330688477, 9.722746849060059, 17.517728805541992, 27.553741455078125, -19.970016479492188, -2.9637999534606934, 11.968571662902832, -29.63854217529297, -13.101225852966309, 37.05523681640625, 32.46376037597656, -10.366905212402344, 13.934648513793945, 27.98902702331543, -12.51738452911377, 14.701377868652344, 53.65060043334961, 23.79286766052246, -27.264814376831055, -18.231313705444336, 2.396273612976074, 6.883164882659912, -13.46916389465332, 11.371989250183105, -0.165763258934021, 15.709270477294922, -47.95379638671875, -32.82372283935547, 22.23992347717285, 2.7282116413116455, 45.987205505371094, -34.000701904296875, -21.234615325927734, 40.11308670043945, 72.6962890625, -21.46512794494629, 28.93805694580078, 43.2562141418457, -7.911442756652832, 52.36796951293945, 17.691972732543945, -30.770036697387695, -41.17550277709961, -4.100633144378662, 16.536481857299805, -26.65749168395996, 38.71052551269531, 26.66131591796875, 10.620739936828613, -5.340798854827881, -0.09476259350776672, 14.39016342163086, 39.03490447998047, 4.8210625648498535, 13.790366172790527, 14.666114807128906, -52.4869384765625, 14.118288040161133, 40.48594284057617, 16.034746170043945, -10.231163024902344, 24.067731857299805, 9.806656837463379, -3.845794200897217, -16.5290584564209, -29.606517791748047, 14.97299861907959, 28.678598403930664, 9.461004257202148, -12.586346626281738, -9.904298782348633, -53.95799255371094, -58.92244338989258, 19.19432830810547, 15.425934791564941, -34.73243713378906, 30.504140853881836, 26.673046112060547, -15.275976181030273, -1.8664418458938599, 20.92400360107422, -36.966033935546875, -20.855852127075195, 21.671707153320312, -11.193774223327637, -7.093689441680908, -14.608065605163574, -26.9197940826416, -13.502049446105957, -29.846288681030273, -83.79769897460938, -32.50702667236328, 6.081081867218018, 11.680254936218262, 65.01786041259766, -18.858688354492188, -28.684314727783203, 1.7239556312561035, -24.700069427490234, 41.63754653930664, 29.603757858276367, -16.557493209838867, -38.42232894897461, -12.27534294128418, 36.68122482299805, -0.6338196992874146, 13.686511039733887, -36.28546905517578, -39.928916931152344, 24.690746307373047, 19.943540573120117, -21.8654727935791, -45.508243560791016, 32.197364807128906, -19.430925369262695, -3.3635635375976562, 24.79703712463379, 26.08060646057129, 62.69320297241211, 5.474780082702637, -24.52059555053711, -1.5307782888412476, 42.29436111450195, 38.82863235473633, 42.3322639465332, 16.53391456604004, 45.646793365478516, -53.95155334472656, 16.51878547668457, -34.58991622924805, 14.892863273620605, 39.05669403076172, 35.83428192138672, 5.8185930252075195, 11.90942668914795, -17.368083953857422, -13.648759841918945, 4.6472859382629395, -25.07472801208496, 49.41549301147461, 10.592386245727539, -15.385199546813965, -8.213748931884766, 42.85933303833008, 15.431924819946289, -26.957605361938477, 11.107269287109375, 19.329870223999023, -12.218749046325684, -13.977932929992676, -24.10822296142578, 36.620208740234375, -14.080951690673828, 13.56804084777832, 39.592628479003906, -23.371023178100586, -5.288325309753418, 28.24182891845703, 13.32969856262207, 87.8900375366211, -13.373661994934082, -34.622371673583984, 5.844075679779053, -0.9187490344047546, 33.27236557006836, 9.740693092346191, 32.37505340576172, -2.353656768798828, 58.75358963012695, -2.172450065612793, 23.386709213256836, -29.84465789794922, -17.4153995513916, 2.3735127449035645, 21.206289291381836, -50.29935073852539, -46.507354736328125, 24.81560707092285, -3.2562456130981445, -28.616212844848633, 11.32789134979248, -2.120833396911621, -35.87018966674805, 16.157665252685547, -15.653992652893066, -45.238494873046875, 2.1083667278289795, 26.19037437438965, 34.83667755126953, -7.19280481338501, -35.731590270996094, 62.91650390625, -12.08468246459961, 2.010270595550537, 42.674598693847656, -37.151580810546875, -9.194449424743652, 7.964960098266602, 0.38642618060112, -35.98555374145508, -5.263372898101807, 53.201988220214844, 0.3789951503276825, -52.01512145996094, -40.00737762451172, -4.152462959289551, 11.57177734375, 5.111275672912598, -27.34588050842285, 53.770503997802734, 54.85541915893555, -18.329483032226562, 16.550729751586914, 41.67869186401367, -5.272675037384033, 17.419788360595703, 13.463672637939453, 36.79336166381836, -11.095978736877441, 16.818008422851562, 42.72135925292969, -21.939355850219727, -15.142053604125977, 22.29840660095215, 10.255188941955566, 15.96005916595459, -49.95725631713867, -35.56471252441406, 23.21324920654297, -26.15072250366211, -47.50627136230469, -61.027130126953125, 20.61612319946289, 19.683767318725586, 71.07010650634766, 22.877864837646484, 14.645051956176758, 7.016861438751221, -26.981121063232422, -9.824989318847656, 28.58800506591797, -7.00416374206543, 7.029900074005127, 10.504260063171387, -34.3724250793457, 8.29409408569336, 20.48118019104004, -0.0560804083943367, -4.720129489898682, 19.051511764526367, -68.1810073852539, 27.73908042907715, 26.70714569091797, 40.728675842285156, 38.07967758178711, -60.62295913696289, 16.213319778442383, 21.171415328979492, -58.718318939208984, -48.84537887573242, 32.05405044555664, 40.56486511230469, -5.154973983764648, 28.05694007873535, -23.755962371826172, 9.677946090698242, -49.64517593383789, 13.454300880432129, -1.1527438163757324, 33.025001525878906, -26.483835220336914, -0.2424297034740448, -33.36943817138672, 0.590869665145874, -23.527544021606445, -47.94759750366211, 2.227593421936035, 21.14330291748047, -16.272979736328125, -42.47555923461914, -15.09467887878418, -39.485008239746094, -15.78368091583252, 7.546144008636475, 30.31173324584961, 5.986617088317871, -17.530193328857422, 3.7580792903900146, -57.934715270996094, -7.433704376220703, 20.520915985107422, -5.941218376159668, 3.304694652557373, -45.36418151855469, -57.58203887939453, -5.25554084777832, 40.6758918762207, -26.032716751098633, -20.088924407958984, -47.39701461791992, 39.54491424560547, 12.306586265563965, -4.511992931365967, -72.82789611816406, 7.233323097229004, 12.522382736206055, 25.38629722595215, -10.072369575500488, 52.10816955566406, 47.659934997558594, -4.7014946937561035, 12.17672061920166, 33.7296257019043, -36.1082763671875, 60.66252136230469, -17.375961303710938, -13.016708374023438, -8.864452362060547, 12.076149940490723, 13.276758193969727, 22.06688690185547, -57.944732666015625, -23.056255340576172, -66.75286865234375, -90.52558898925781, 35.311279296875, -15.042364120483398, -28.332334518432617, -12.3411865234375, -27.266416549682617, -28.190052032470703, 25.579063415527344, 12.353110313415527, 42.10797119140625, -0.45416709780693054, -5.923642158508301, -3.3748655319213867, -39.30464172363281, -21.3166446685791, 33.668373107910156, -4.940821170806885, -42.675533294677734, -63.55455017089844, 38.989227294921875, 25.418317794799805, -50.49259567260742, -9.250685691833496, 28.018142700195312, 29.963592529296875, 13.673822402954102, -12.91213607788086, -38.3687744140625, 64.60491180419922, 62.08818817138672, 40.42490005493164, -19.53070640563965, 16.955467224121094, -20.889015197753906, 2.596405506134033, 9.2256441116333, 50.642364501953125, -23.59627914428711, -24.497346878051758, -16.191659927368164, 18.404342651367188, -35.497222900390625, 0.21137584745883942, -31.840267181396484, 18.50568389892578, -7.297909736633301, 25.058353424072266, 30.155799865722656, 12.501273155212402, 52.9558219909668, 25.384859085083008, 77.01994323730469, 35.510231018066406, -16.314485549926758, -28.689071655273438, 25.9088191986084, 14.75110149383545, -6.206335067749023, -13.124373435974121, 12.4923095703125, 15.382089614868164, -33.99959945678711, 1.4766786098480225, -1.1046212911605835, -20.306272506713867, -38.46176528930664, 46.239349365234375, 37.08724594116211, -40.137699127197266, -13.381360054016113, -12.953703880310059, -19.727136611938477, 24.772329330444336, -30.19188117980957, -19.562124252319336, -12.603865623474121, 2.7790377140045166, 30.7193603515625, 8.199138641357422, 16.882244110107422, -60.699005126953125, -18.419389724731445, 7.06791353225708, 0.13512510061264038, -12.115772247314453, -17.346660614013672, -24.602882385253906, -49.450927734375, -18.272329330444336, 33.66132354736328, 4.59669303894043, -37.6396484375, -13.614937782287598, 4.90336275100708, 51.57144546508789, -54.6939811706543, 0.8426083326339722, 5.9989447593688965, 2.6626029014587402, 48.21527099609375, 5.703754425048828, 48.32566833496094, -3.6528007984161377, 39.76216506958008, 14.224395751953125, -15.766997337341309, -4.669312953948975, 17.051122665405273, 26.54080581665039, 10.548493385314941, 14.869407653808594, 40.64729690551758, -14.0032377243042, -51.31708908081055, -1.8108464479446411, 30.877267837524414, -3.6545746326446533, 7.924808979034424, -22.54912567138672, -10.207758903503418, 0.3933814465999603, 49.90209197998047, 3.6131060123443604, 42.15635299682617, -7.877549648284912, -20.014474868774414, 4.931450843811035, 12.855469703674316, 54.509525299072266, -67.7386703491211, 11.694108963012695, -34.74928283691406, -60.43213653564453, 8.744735717773438, -10.561641693115234, -7.433305740356445, -10.63719367980957, -41.61980438232422, 39.905052185058594, 27.537311553955078, 30.746826171875, 2.9235124588012695, -3.1221766471862793, 10.550684928894043, 11.430858612060547, 9.388082504272461, -23.235153198242188, 17.594131469726562, 40.15676498413086, -0.6414368748664856, 24.02922821044922, 37.11540985107422, -13.54283618927002, -7.681733131408691, 16.899084091186523, -35.394046783447266, 40.99186325073242, -12.732169151306152, 11.898337364196777, 0.7199422717094421, 4.503332614898682, -4.560523986816406, -8.883124351501465, 24.99390411376953, 7.477778434753418, 17.543930053710938, -18.299184799194336, 37.055503845214844, -28.342443466186523, 14.826432228088379, -33.70177459716797, -1.4562686681747437, -43.70460510253906, -6.950953483581543, -10.733288764953613, -37.054595947265625, 15.949319839477539, -4.360869884490967, 28.574127197265625, -0.4100189805030823, -8.192659378051758, 27.560842514038086, -51.78078842163086, -20.68274688720703, 34.117610931396484, 13.98798942565918, 13.920148849487305, 8.296995162963867, -3.526700019836426, -65.88084411621094, 11.915596961975098, -11.190108299255371, -23.05016326904297, -26.14836883544922, -24.113555908203125, -13.78847885131836, -32.71352005004883, -41.49945068359375, 39.359439849853516, -36.072818756103516, -24.001541137695312, 9.90804386138916, 9.61423397064209, -21.983829498291016, -1.840051293373108, -11.695440292358398, -5.8609137535095215, -33.953155517578125, 0.3685021996498108, -1.089453101158142, 13.30477523803711, -29.391094207763672, 41.25284194946289, -27.293107986450195, 10.226816177368164, 41.77033615112305, 31.46347427368164, 6.83945369720459, 7.3706488609313965, -67.7701187133789, -35.23417663574219, -5.635032653808594, 1.9485325813293457, -48.395118713378906, 17.00954818725586, 9.029200553894043, -22.39665985107422, -0.07475940138101578, 48.07014083862305, 23.304264068603516, 8.027222633361816, -7.163035869598389, -30.138612747192383, 16.462432861328125, -29.570547103881836, 18.984865188598633, -16.564926147460938, 9.731400489807129, -57.70142364501953, -3.417574405670166, 33.994354248046875, 20.88726806640625, -13.352102279663086, -18.17646026611328, -8.782209396362305, 6.457630634307861, -8.417980194091797, 1.6662741899490356, -15.117890357971191, -29.101886749267578, 55.885414123535156, -11.515256881713867, 3.9545252323150635, 49.15532684326172, -12.507109642028809, -3.4802024364471436, 3.805361747741699, -5.022187232971191, -36.84743881225586, 39.10893630981445, -12.461612701416016, 3.831965684890747, 34.22772216796875, 28.751480102539062, 21.93212890625, -56.17193603515625, 19.082677841186523, 51.90008544921875, 1.9003195762634277, 11.270155906677246, 8.095824241638184, -38.5737190246582, -47.54505157470703, -47.032203674316406, 5.423092365264893, -37.71086883544922, -15.231369018554688, -13.617913246154785, 44.6590690612793, 7.512838840484619, -16.905187606811523, -16.068769454956055, 24.681989669799805, -31.14322853088379, -57.608177185058594, -24.83238410949707, -11.001069068908691, 23.96158218383789, -5.707103252410889, 1.4937355518341064, -19.993194580078125, -16.09273910522461, 52.973548889160156, -53.19328308105469, 5.372906684875488, 1.6384003162384033, 59.255496978759766, -21.618539810180664, -43.888465881347656, -2.0924489498138428, 35.48395919799805, 20.05251121520996, -40.70174789428711, -16.29627227783203, -15.141305923461914, 18.358285903930664, 51.45417022705078, 3.2190988063812256, -38.1556396484375, 59.02918243408203, -7.405953407287598, -40.75014114379883, -50.683677673339844, -24.883281707763672, 27.957429885864258, -9.963709831237793, 51.751102447509766, -11.608515739440918, 1.070183515548706, -29.37302589416504, -1.7252861261367798, -11.229697227478027, -2.714402437210083, -13.070645332336426, 31.288448333740234, 35.403411865234375, -25.937915802001953, -51.43959426879883, -0.9533559679985046, -19.617536544799805, -1.300889253616333, 39.97240447998047, -8.3785982131958, 12.271048545837402, 21.174697875976562, 49.17029571533203, 20.847719192504883, -25.068635940551758, -29.27413558959961, -27.8323917388916, 3.671248197555542, -14.341381072998047, -13.719205856323242, -7.793882369995117, 2.184190511703491, -13.587616920471191, -41.118064880371094, 5.837760925292969, 34.283443450927734, 39.14093017578125, 20.17542839050293, 70.59272766113281, -64.66817474365234, -24.848400115966797, 30.878202438354492, 17.489845275878906, 26.046615600585938, 8.801573753356934, 75.44026947021484, 46.78276443481445, 9.016881942749023, -65.96842956542969, -44.790740966796875, 29.098358154296875, 78.02119445800781, -7.875616550445557, 17.125988006591797, -17.14580726623535, -19.77229881286621, 16.011417388916016, 1.45955491065979, -41.72499465942383, -69.09650421142578, -2.9973366260528564, 80.86746215820312, 16.49640464782715, -13.643123626708984, 64.76103210449219, 36.07278823852539, 20.432220458984375, 3.700833320617676, -13.686809539794922, -21.233020782470703, 1.60193932056427, -8.602899551391602, 6.973763465881348, -25.928955078125, -34.90633773803711, 9.879509925842285, 9.22329330444336, 37.514827728271484, -8.764571189880371, -22.965469360351562, 21.567520141601562, -16.49141502380371, 11.16547679901123, 38.17485809326172, -18.297943115234375, -12.822125434875488, -11.778398513793945, -11.655660629272461, 7.136530876159668, 13.967055320739746, -3.399040460586548, 9.16835880279541, -6.140264511108398, -49.70368957519531, -11.039562225341797, 26.21432876586914, -24.7576904296875, -22.189971923828125, -3.2292404174804688, 12.905299186706543, 8.230015754699707, 21.634546279907227, 48.78587341308594, 11.623881340026855, -10.555312156677246, 2.2139601707458496, 40.491233825683594, -2.131333589553833, -16.62397575378418, 24.613609313964844, 1.3269063234329224, 23.859573364257812, 16.616641998291016, -20.194965362548828, -11.596091270446777, 24.51803970336914, -21.41398048400879, -34.0111198425293, -79.9537353515625, -0.09701687842607498, -14.088984489440918, 0.6502393484115601, 6.493447303771973, -49.90126419067383, 26.25183868408203, -41.19606399536133, 15.598470687866211, -0.2601058781147003, -6.833363056182861, -27.844268798828125, 39.67607116699219, 7.254420280456543, -25.340017318725586, 17.387434005737305, 9.94947624206543, -10.540916442871094, 6.043522834777832, -14.782512664794922, -18.34535789489746, 14.445427894592285, 3.645282506942749, -8.30941390991211, -27.7789363861084, -21.089256286621094, 16.532743453979492, -57.07261276245117, -18.27044105529785, -35.59825134277344, 0.8013078570365906, -54.480140686035156, 34.69256591796875, -17.175495147705078, 21.528785705566406, 23.53235626220703, 0.7727037668228149, 8.598230361938477, 31.656085968017578, 19.805421829223633, -6.781398296356201, -6.98390531539917, -25.44732666015625, -16.439115524291992, 20.406362533569336, 0.8017260432243347, -13.204809188842773, -53.9450569152832, 39.80135726928711, 6.764020919799805, -37.38710021972656, -55.82231140136719, -6.964334487915039, 20.89691162109375, 30.617050170898438, -30.406003952026367, 26.109182357788086, -48.77767562866211, 17.158885955810547, 0.49596649408340454, 4.375724792480469, -50.37998580932617, -39.21011734008789, -14.666394233703613, -3.318040370941162, -12.99250316619873, 14.86680793762207, 24.771713256835938, 16.49178123474121, 76.21578216552734, -15.499898910522461, -46.68259811401367, 10.776782989501953, -8.784661293029785, 41.95463943481445, -11.04195499420166, 4.605649471282959, 3.6620664596557617, -11.354141235351562, 9.124336242675781, 66.39270782470703, 35.972660064697266, -68.90898132324219, 9.686131477355957, -19.6201171875, -67.74835968017578, 40.9915771484375, -23.15324592590332, -5.7092390060424805, -66.8957748413086, -27.3420467376709, -78.09091186523438, -3.808764934539795, -15.09803295135498, 45.17431640625, 23.11643409729004, 4.404338836669922, -41.273414611816406, -6.498627662658691, 4.5922956466674805, -9.374428749084473, 22.231752395629883, -21.39940643310547, 13.382954597473145, 7.857641220092773, 72.1036376953125, 7.262012481689453, 16.576108932495117, -47.06111145019531, -25.822799682617188, 7.115777015686035, -18.225130081176758, -23.120826721191406, 9.529762268066406, -41.21928405761719, -8.293299674987793, 24.85002326965332, -6.258877754211426, -41.79336929321289, 20.04022216796875, -14.596567153930664, -12.313813209533691, -21.722631454467773, 4.376874923706055, 2.3433454036712646, 6.431448459625244, 32.683555603027344, 22.590383529663086, -7.540991306304932, 2.7491722106933594, 0.23467275500297546, -21.20579719543457, -17.843334197998047, 13.75734806060791, -37.300113677978516, 34.07752227783203, 66.3460922241211, -35.85631561279297, -9.889266014099121, -10.339241027832031, 4.527463436126709, -34.97388458251953, -27.026296615600586, 0.35498446226119995, 3.0517959594726562, -26.102880477905273, 37.43495178222656, 20.047149658203125, 44.482234954833984, -55.55746841430664, -39.366085052490234, 63.94718933105469, 0.43542295694351196, 40.22453308105469, 28.468364715576172, -31.71078109741211, 9.767401695251465, 17.499370574951172, -8.288232803344727, 5.4326066970825195, -4.540011405944824, 19.0082950592041, 41.55707550048828, -38.35369873046875, -1.077887773513794, -46.05646514892578, -28.508384704589844, 3.6115572452545166, -52.4948616027832, 10.194710731506348, -39.867244720458984, 40.841800689697266, 5.365197658538818, 10.599401473999023, 16.448022842407227, 13.659671783447266, 16.708045959472656, 12.940937042236328, -49.844940185546875, 17.110612869262695, 43.799625396728516, -34.349727630615234, 1.774385929107666, -43.92097473144531, -18.117050170898438, -17.566259384155273, -5.872813701629639, -19.338163375854492, 24.13277244567871, -21.66302490234375, -23.56228256225586, -9.321447372436523, 26.139263153076172, 33.944313049316406, -35.65791702270508, 15.407144546508789, -0.13024070858955383, 23.390104293823242, -24.24839973449707, 7.504278182983398, -2.830510139465332, -6.54400634765625, 24.405941009521484, 26.25588607788086, -13.245184898376465, -8.181341171264648, 13.66769790649414, -20.852441787719727, -51.37442398071289, -17.9165096282959, -9.415657043457031, -14.002172470092773, -42.970184326171875, 6.551537990570068, -7.445644855499268, -9.64240550994873, -31.176298141479492, 42.48667907714844, 48.491600036621094, 7.593493938446045, 8.53183650970459, -7.843954086303711, -6.045032501220703, 19.911775588989258, -15.767753601074219, 16.04213523864746, 29.543380737304688, -13.281248092651367, 54.57748031616211, 0.36964258551597595, 56.910037994384766, 32.440826416015625, -21.534875869750977, 41.091312408447266, 13.174450874328613, 47.16560745239258, 7.91347074508667, 40.803733825683594, -61.59337615966797, -14.083100318908691, -21.625385284423828, 13.402097702026367, -29.167957305908203, -21.03598403930664, 5.071614742279053, 17.4226016998291, -45.101470947265625, 33.14872741699219, -49.35409164428711, 3.582016706466675, 25.923994064331055, -28.489479064941406, -1.5593442916870117, -18.615026473999023, -7.350047588348389, -8.790018081665039, 7.382593154907227, -2.198662281036377, 19.94614028930664, 60.42439651489258, 7.3017377853393555, 18.30055809020996, -1.7308682203292847, -9.24484920501709, -72.77764892578125, -17.665042877197266, 14.487630844116211, -18.36004066467285, -13.323787689208984, -1.9639644622802734, 3.2856643199920654, -8.10167121887207, -65.3775634765625, -24.597129821777344, 7.7483062744140625, 2.9057328701019287, -32.935855865478516, 24.641380310058594, 11.515052795410156, -18.095266342163086, 29.518177032470703, 32.996421813964844, -19.47719955444336, 2.0900776386260986, 36.75736618041992, 17.290117263793945, 32.1055908203125, -67.89691162109375, -25.977388381958008, 30.087078094482422, 11.81725788116455, 4.835402488708496, -14.184049606323242, -5.716034412384033, -32.100425720214844, -55.54491424560547, -19.155593872070312, 4.501558780670166, 17.79913902282715, -0.8195250034332275, 12.184653282165527, -12.28099250793457, 19.46725082397461, 48.08525085449219, 34.33952713012695, 42.5724983215332, 14.65255069732666, -55.485660552978516, -17.117656707763672, 35.56757354736328, 36.772743225097656, 13.009603500366211, 37.261436462402344, -51.64341354370117, 49.7038688659668, 32.888240814208984, -56.973365783691406, -35.66154861450195, -38.08378601074219, 37.193546295166016, -24.65546226501465, 12.132701873779297, -55.714622497558594, 20.230552673339844, 27.259151458740234, 8.96251106262207, 8.82531452178955, -27.903884887695312, 10.94227409362793, -4.684534072875977, -11.620534896850586, -8.293380737304688, 29.61811637878418, 12.473834991455078, -18.357440948486328, 38.57688903808594, -79.99834442138672, 3.747407913208008, -29.10374641418457, 64.66653442382812, 12.626039505004883, -1.3415199518203735, -20.68779182434082, 51.422054290771484, 40.22191619873047, 32.51649475097656, -10.765625953674316, 7.481478691101074, -4.480066299438477, -31.779645919799805, -33.90692901611328, 2.523782968521118, 37.128883361816406, 1.9817373752593994, 18.041555404663086, -23.889842987060547, 30.380741119384766, -19.714731216430664, 29.384384155273438, 4.04160213470459, -29.535871505737305, 46.446876525878906, -27.22788429260254, -6.811731815338135, -24.480321884155273, 59.53611373901367, 6.892368316650391, 15.45666217803955, 16.662464141845703, -7.6446533203125, -7.733239650726318, 54.11558532714844, -19.714242935180664, 58.31556701660156, -42.573543548583984, 5.580521583557129, 2.9571189880371094, 15.49488353729248, 0.15692982077598572, -9.258512496948242, 13.534662246704102, -14.124297142028809, -36.708683013916016, -11.173274993896484, 12.219209671020508, 2.7663557529449463, 0.8610393404960632, 20.567686080932617, -31.05030632019043, 3.190692186355591, -7.307711601257324, 5.419167518615723, -30.084796905517578, 19.022197723388672, 16.897850036621094, -3.2787981033325195, -50.18549346923828, 69.37307739257812, -49.188568115234375, 15.828954696655273, 4.682173252105713, -23.55474853515625, -4.015852451324463, -27.33416748046875, -3.7038447856903076, 13.595635414123535, -7.403316974639893, -17.743125915527344, -7.315086841583252, 36.76849365234375, -53.01285934448242, -37.15338897705078, 16.120588302612305, -51.27182388305664, -8.526510238647461, -11.757389068603516, 38.647865295410156, -11.889117240905762, -7.282182216644287, 14.675012588500977, 14.228387832641602, -56.58283996582031, 18.531736373901367, -10.693424224853516, 46.04590606689453, -12.216136932373047, 15.048741340637207, 31.786151885986328, -27.364654541015625, 24.746944427490234, -16.808589935302734, 53.382080078125, -16.22918128967285, 42.0502815246582, 23.330726623535156, -18.314428329467773, -22.44307518005371, -7.172585964202881, 37.27013397216797, -16.225156784057617, 4.140504837036133, 9.242012023925781, -11.96603012084961, 13.795616149902344, 67.92252349853516, -2.8347792625427246, 47.30628204345703, 28.985868453979492, 42.36903381347656, -4.4002180099487305, 3.0853419303894043, -35.05919647216797, 20.98224449157715, -0.6557579636573792, 21.174480438232422, 8.087170600891113, 37.56330490112305, 20.228416442871094, -0.36164340376853943, 33.687625885009766, -39.344966888427734, 41.998897552490234, 1.3943901062011719, -48.54448318481445, -7.926604270935059, 28.180137634277344, -17.57485580444336, -4.565819263458252, 26.890493392944336, 82.14824676513672, 18.60251808166504, 43.630916595458984, 53.79692459106445, -65.53682708740234, -33.67549133300781, 10.7493896484375, -54.49623107910156, -51.64863586425781, 34.416786193847656, -25.303247451782227, 15.16100025177002, 9.055280685424805, -4.321899890899658, 2.5604209899902344, -9.888331413269043, -52.13915252685547, -44.846317291259766, 41.016300201416016, -10.48563003540039, 19.02853012084961, 15.053524017333984, 19.042112350463867, 22.111921310424805, 59.533966064453125, 31.609582901000977, -3.1156508922576904, 45.123565673828125, 3.3018686771392822, -11.827003479003906, -10.688216209411621, -2.3785400390625, 51.426246643066406, 19.80527687072754, 46.80736541748047, 53.59336853027344, -44.8100700378418, 29.877277374267578, -7.090599060058594, -33.397212982177734, -13.725558280944824, 33.30131530761719, -4.289378643035889, -11.317870140075684, -29.336252212524414, -18.476242065429688, 30.823699951171875, -43.427188873291016, -5.35360860824585, -11.301466941833496, 8.89683723449707, -38.19234085083008, -19.69261360168457, -20.7678279876709, 4.153985500335693, 26.907062530517578, 14.082776069641113, -60.9839973449707, -0.561581015586853, 23.63094139099121, -19.341272354125977, -22.131275177001953, -36.1314811706543, 13.870401382446289, -13.413675308227539, -76.17605590820312, -27.8315372467041, 3.4652907848358154, 34.462738037109375, -10.088017463684082, -31.715490341186523, 42.04150390625, -25.94866943359375, 42.09280014038086, 2.2216835021972656, 0.36604177951812744, -31.920818328857422, 2.7045018672943115, 32.5179328918457, -23.742874145507812, 2.567349433898926, -4.755881309509277, -11.736838340759277, -27.870590209960938, -3.6527748107910156, -13.207314491271973, 18.980926513671875, -19.247699737548828, -3.7146217823028564, 13.32812786102295, -2.0637567043304443, -36.247398376464844, -39.94573974609375, -32.29946517944336, 10.717838287353516, -31.019865036010742, -38.595008850097656, 66.2038803100586, 19.446327209472656, 35.246063232421875, -66.67768096923828, -28.252155303955078, -5.138762474060059, -23.39101791381836, 16.710973739624023, 13.316862106323242, 15.335894584655762, 33.442501068115234, -14.97093677520752, 13.240608215332031, 37.061622619628906, 6.133912563323975, 8.275907516479492, -7.773852825164795, 21.576032638549805, -7.690402984619141, 18.917430877685547, -9.84630012512207, -22.699623107910156, 56.04172134399414, 3.0454983711242676, -12.403200149536133, -40.22316360473633, -20.91082000732422, 61.237022399902344, 40.41930389404297, 33.902061462402344, 9.559043884277344, 1.3804957866668701, 53.67856216430664, -28.827552795410156, 6.2060041427612305, 22.852216720581055, -26.05544662475586, -26.44437026977539, -6.011582374572754, 13.356934547424316, -10.916187286376953, 12.680362701416016, -52.1419563293457, 29.309202194213867, -7.5517191886901855, -18.446870803833008, 59.90586853027344, -8.06867790222168, -18.742408752441406, 22.80697250366211, -9.45026969909668, -37.83877944946289, -15.323837280273438, -14.605056762695312, 12.830718994140625, -1.9308959245681763, -60.111793518066406, -2.1300840377807617, 21.976877212524414, 0.2329479604959488, -45.184844970703125, -19.693445205688477, -2.006753444671631, -16.01850700378418, 41.47715377807617, 2.278491973876953, -9.058409690856934, -34.83392333984375, -5.983863830566406, -15.508275032043457, 25.788362503051758, 21.686479568481445, -28.859580993652344, 32.646400451660156, -7.860921859741211, 27.211807250976562, 10.246002197265625, -2.6584906578063965, -2.43464732170105, -22.65041160583496, -8.85403823852539, 23.992204666137695, -24.99690055847168, 2.481961727142334, -35.05485153198242, -32.41435241699219, -24.042362213134766, -59.14464569091797, 24.002696990966797, 8.403818130493164, -6.869631767272949, 8.680719375610352, 38.50310516357422, -16.475303649902344, -46.469322204589844, -12.917757987976074, -5.487982749938965, 2.535397529602051, -3.1234028339385986, -7.2816972732543945, -75.78706359863281, -32.33348083496094, -4.986868381500244, -24.209041595458984, 72.7010726928711, 45.28667068481445, 16.386018753051758, 3.3431308269500732, 34.13804244995117, 2.786750555038452, 15.028036117553711, -5.610588073730469, 9.77839469909668, -13.460539817810059, -9.481368064880371, 63.93988800048828, -2.3136203289031982, -3.105187177658081, -20.341161727905273, 44.35944366455078, -34.340118408203125, 21.569772720336914, -7.106608867645264, 5.632910251617432, -20.467775344848633, 0.57671719789505, 14.367351531982422, -1.8744397163391113, -7.632122993469238, -25.02933120727539, 9.756608963012695, -30.072919845581055, 9.891843795776367, 35.66091537475586, 18.858816146850586, -25.122528076171875, 1.9518424272537231, -18.494426727294922, 15.26459789276123, 17.06540298461914, -46.49268341064453, 2.053757429122925, -18.28415870666504, -24.11687660217285, -24.9233341217041, 13.794315338134766, -2.9549617767333984, 7.046119213104248, 16.73217010498047, -5.723291397094727, -51.72972106933594, 17.34523582458496, -23.609540939331055, 17.775081634521484, -19.56980323791504, -7.072895526885986, -48.07758712768555, 53.30010986328125, 7.384604454040527, -24.766206741333008, -12.298215866088867, -17.199045181274414, 19.519250869750977, -30.715877532958984, -11.904881477355957, 22.72123146057129, -28.57522201538086, 26.32538414001465, 0.23410730063915253, 65.38919067382812, 11.398213386535645, 19.1573429107666, -12.901423454284668, -9.763540267944336, 3.032918930053711, -28.132734298706055, -40.3287353515625, -29.181856155395508, -1.056919813156128, -12.241277694702148, -33.28068161010742, 7.26084041595459, 2.59185791015625, 15.291068077087402, -35.485862731933594, -4.6261701583862305, -12.854316711425781, -22.484638214111328, 1.835795283317566, 36.219032287597656, 1.2502710819244385, 7.667313575744629, 49.73582077026367, -27.485246658325195, 22.5985107421875, -26.201967239379883, 25.016244888305664, -46.321834564208984, 28.01526641845703, -0.2978118360042572, 17.426162719726562, -18.589290618896484, -36.380126953125, -82.32100677490234, -8.089743614196777, -38.96147918701172, -21.31669807434082, 9.957463264465332, -35.07324981689453, -1.917832374572754, -11.890127182006836, 17.80095100402832, 21.371967315673828, 41.2468147277832, -25.843578338623047, -5.0618486404418945, 44.1373291015625, 28.28057289123535, 27.51043701171875, 13.27813720703125, 41.94405746459961, -15.162955284118652, 5.333832263946533, 49.21477508544922, -53.68870544433594, -23.819263458251953, 28.336462020874023, -3.1216847896575928, -41.1989860534668, -46.931575775146484, 18.470613479614258, -23.7273006439209, -15.698410034179688, 3.0058705806732178, 6.999873638153076, -18.254112243652344, -91.18110656738281, -3.9067938327789307, 34.741065979003906, -35.316715240478516, -17.99301528930664, 21.280881881713867, 3.458930253982544, 4.817661285400391, 2.11442494392395, 50.83165740966797, -16.53087043762207, -11.472821235656738, 50.140621185302734, 51.421852111816406, 28.74642562866211, 31.9011173248291, -33.101234436035156, -0.10638734698295593, -25.44179344177246, 43.0505485534668, 4.862825393676758, -30.669279098510742, -63.09056854248047, -18.418405532836914, 44.69231414794922, -11.435297966003418, -11.042525291442871, -46.95287322998047, -16.23029899597168, -75.34635925292969, 2.049086093902588, -1.3033729791641235, -4.406093597412109, 18.12489128112793, 22.55033302307129, -62.831504821777344, -17.484344482421875, -47.25959777832031, -53.777305603027344, 20.331192016601562, 13.063838958740234, -20.058992385864258, -39.03270721435547, -5.751819133758545, -3.1496877670288086, -20.164247512817383, 55.339263916015625, 1.2368569374084473, -43.65596389770508, 58.72099685668945, -18.350013732910156, -40.112396240234375, 10.530733108520508, -19.47194480895996, 8.544188499450684, -33.46809387207031, 29.3387393951416, -23.593067169189453, -44.28408432006836, 48.64653015136719, -35.9269905090332, 66.2182846069336, 35.937347412109375, -11.708843231201172, -33.64384078979492, 8.840348243713379, -19.863494873046875, -22.67048454284668, 2.321502923965454, -44.344913482666016, -31.808391571044922, 5.551153659820557, 24.167972564697266, -10.693145751953125, 0.6946419477462769, 12.999844551086426, 23.363418579101562, -26.197181701660156, 32.50773239135742, -36.699153900146484, -56.68353271484375, 1.0120611190795898, -44.546531677246094, -27.03251075744629, 6.74976110458374, 25.171566009521484, -18.862220764160156, -0.620628833770752, 9.856386184692383, 5.382667541503906, -8.468033790588379, -14.992116928100586, -1.8629851341247559, -10.455880165100098, -31.697473526000977, -24.530113220214844, -44.876953125, -23.738168716430664, -7.782116413116455, -25.756391525268555, -11.93200969696045, 33.727169036865234, 51.21992111206055, -99.13835906982422, 7.049025535583496, 4.7586140632629395, -8.941259384155273, 41.27110290527344, 39.06887435913086, -22.957311630249023, -3.8736672401428223, 0.7537047863006592, -71.48143768310547, -7.05735969543457, 6.054743766784668, -20.783952713012695, 17.14286231994629, 3.786756992340088, 50.114688873291016, -4.589054584503174, -17.903850555419922, -1.2140098810195923, -0.5005865693092346, 25.78898048400879, 21.586631774902344, -31.384323120117188, 44.979736328125, 26.04042625427246, -63.04909133911133, 11.966370582580566, 22.661033630371094, -3.7665233612060547, -25.850872039794922, -42.298606872558594, 14.745659828186035, 30.10281753540039, -2.525786876678467, 24.318052291870117, 69.77482604980469, -9.201233863830566, 33.21145248413086, 18.365371704101562, 23.958690643310547, -29.099964141845703, -13.972394943237305, 15.918139457702637, 6.003544807434082, -11.334874153137207, -4.4571123123168945, 51.224754333496094, 39.7677116394043, -4.5061726570129395, 20.323095321655273, -19.882844924926758, 11.59924602508545, -24.02983283996582, -3.8894150257110596, 23.673765182495117, -1.7319759130477905, 7.12556266784668, 26.438079833984375, -12.881591796875, 13.416850090026855, -17.27952766418457, 2.630293130874634, 32.74878692626953, 0.20733411610126495, -28.788007736206055, 25.151731491088867, 26.56051254272461, -1.6298311948776245, -12.616495132446289, -8.58144760131836, -7.870182514190674, -3.4359583854675293, -27.43494987487793, 3.923490524291992, -1.6033731698989868, 6.500267028808594, 34.00712966918945, -7.678407192230225, 34.31538391113281, -31.646804809570312, -33.43256378173828, -4.11888313293457, 13.247855186462402, 7.286536693572998, 33.941856384277344, 8.567800521850586, 32.230491638183594, 48.493499755859375, 4.188608169555664, 6.085996627807617, -33.45928192138672, -36.279335021972656, 22.8053035736084, 47.295997619628906, 34.576045989990234, -3.349822998046875, 14.628768920898438, 17.332210540771484, 39.53354263305664, 25.414033889770508, -40.31306838989258, 39.173912048339844, -32.15483474731445, 0.32516419887542725, -18.300107955932617, -27.3959903717041, 16.429574966430664, -56.30574417114258, -17.80687713623047, -8.683647155761719, 4.276042938232422, 23.62143325805664, -4.107180595397949, 17.56940460205078, -37.36745071411133, -16.427207946777344, 21.85781478881836, 7.209737777709961, 41.340911865234375, 31.500823974609375, -2.431234836578369, -46.5657958984375, 5.608373641967773, -49.43230438232422, 20.532833099365234, 15.909784317016602, 15.840846061706543, -17.159759521484375, -15.107826232910156, 35.839962005615234, 24.498167037963867, 29.78826332092285, 25.610523223876953, 54.02664566040039, -6.496776103973389, 0.22260716557502747, -4.39475679397583, 9.754618644714355, 1.7510782480239868, -5.382613182067871, 17.466703414916992, -12.820560455322266, -0.5620569586753845, -6.2390031814575195, -13.562519073486328, 18.75154685974121, -5.081286907196045, 37.82756423950195, -6.689003944396973, 23.326351165771484, -79.57341003417969, 1.6481069326400757, 0.6248531341552734, -11.045822143554688, 36.42042541503906, 11.75449275970459, 25.65734100341797, 3.007075548171997, -27.09491729736328, 4.876393795013428, 7.234970569610596, 37.346553802490234, 47.42430877685547, 24.024206161499023, -8.823732376098633, -86.93408203125, 46.79240417480469, 50.07190704345703, -39.518577575683594, -35.82218933105469, -14.512858390808105, -8.364896774291992, 7.06268310546875, -40.563621520996094, 1.6266180276870728, 29.97503089904785, -14.476516723632812, 22.4505558013916, -11.361502647399902, -11.447856903076172, 67.27128601074219, -60.99211883544922, 30.775476455688477, -59.70050048828125, -35.242401123046875, -30.80508041381836, -24.536151885986328, -0.4021436870098114, 41.64628219604492, 4.840868949890137, 50.063961029052734, 4.3736572265625, -19.732986450195312, 24.946292877197266, 33.41783142089844, 32.83889389038086, -3.431917428970337, 16.32807159423828, 35.5012092590332, -30.20882225036621, 20.462337493896484, 35.833675384521484, 2.115678310394287, -35.66991424560547, 7.478096008300781, 35.14552307128906, -23.6215877532959, -28.92217445373535, -44.247982025146484, 4.977745056152344, 2.4220211505889893, 60.28642272949219, -1.4020349979400635, 29.14946174621582, -15.984823226928711, -11.90329647064209, 26.202486038208008, 14.511226654052734, 49.735469818115234, 0.833492636680603, -23.963993072509766, -34.356719970703125, -13.058798789978027, -15.412230491638184, -32.74991989135742, 40.88531494140625, 72.65994262695312, -32.82695007324219, -8.989136695861816, -91.10536193847656, 23.3013858795166, 17.823162078857422, 32.5975341796875, -4.370338439941406, 31.421445846557617, -6.422622203826904, -44.69856643676758, 41.62666320800781, 33.21894454956055, -28.518390655517578, -17.42257308959961, 45.94050979614258, -45.178165435791016, -32.550636291503906, 6.155313968658447, 9.703998565673828, 40.75874710083008, -0.5301560163497925, -8.712605476379395, -5.709200859069824, 30.111391067504883, 22.56138038635254, -11.024029731750488, -59.71038055419922, -16.418519973754883, 8.900390625, -48.803672790527344, 56.44692611694336, -28.740676879882812, 16.63287353515625, 7.600009918212891, -18.781723022460938, 54.35557556152344, -9.022550582885742, 32.2807731628418, -0.08564151078462601, -23.710369110107422, -1.367619514465332, -15.310760498046875, -19.1004581451416, 18.4837646484375, 6.7299957275390625, 24.24892807006836, -3.234764814376831, -46.7949104309082, 5.704280376434326, 28.089975357055664, -7.382843971252441, 32.62346649169922, -8.322625160217285, 8.643847465515137, -6.063417434692383, 1.0933382511138916, 3.2644543647766113, 9.869300842285156, -3.5876333713531494, -5.895445346832275, -18.32251739501953, -44.37344741821289, -35.56075668334961, -0.9550293684005737, -92.76640319824219, 4.8088059425354, 52.6933479309082, 41.030418395996094, 23.967205047607422, 17.655128479003906, -23.403839111328125, 6.072325229644775, -24.272953033447266, -8.111259460449219, 13.522092819213867, 37.47789001464844, 7.703404426574707, 33.988433837890625, -15.79898738861084, -50.71581268310547, 9.006705284118652, -30.592500686645508, 31.006511688232422, -41.67817306518555, -3.406928300857544, 35.54116439819336, 14.862133026123047, 22.654190063476562, 54.88543701171875, 32.0125617980957, 56.406497955322266, -14.912931442260742, 24.458698272705078, -0.38842031359672546, -12.057308197021484, 46.82107925415039, -48.21416091918945, 34.3111572265625, 36.83306884765625, -3.613293170928955, -1.6004700660705566, -1.1733430624008179, 14.228504180908203, -43.3466911315918, 11.233988761901855, 9.763883590698242, -32.89608383178711, -1.4167560338974, -10.522588729858398, -20.320589065551758, -20.977983474731445, -11.906455993652344, 21.576927185058594, -27.57337188720703, 4.868984222412109, 42.29477310180664, -34.48550796508789, 3.5320627689361572, 27.130172729492188, -29.710403442382812, -13.624146461486816, -24.70880889892578, -31.46766471862793, -42.38594055175781, -8.031646728515625, -9.155157089233398, -68.56290435791016, 8.069002151489258, 44.94953155517578, 36.194271087646484, -51.776451110839844, -2.942910671234131, -53.050376892089844, 11.603564262390137, -49.405799865722656, 7.193874359130859, -42.357452392578125, -58.46430969238281, -114.10911560058594, -13.345952987670898, -9.345394134521484, 22.829877853393555, -29.239192962646484, 18.582496643066406, -9.147188186645508, -29.226139068603516, -7.3959455490112305, -34.114990234375, 34.908138275146484, -13.572624206542969, 36.150047302246094, 47.28969955444336, -2.458519458770752, 17.63799285888672, -23.479257583618164, 17.08991813659668, 46.804447174072266, 17.004695892333984, -16.435333251953125, 24.875394821166992, 32.00021743774414, 4.770294189453125, -0.9730265140533447, -67.51261901855469, 2.082853317260742, 28.57692527770996, 20.52442741394043, -2.602743148803711, -13.026519775390625, -52.37571716308594, 40.82511520385742, 16.384838104248047, 3.3598480224609375, -10.668010711669922, 0.05238161236047745, 40.28032684326172, 5.590939998626709, -10.859583854675293, -36.4759407043457, -9.218103408813477, 43.958133697509766, -27.28751564025879, -17.881595611572266, 41.398658752441406, -16.90484046936035, 8.54541015625, 11.898088455200195, -19.11431884765625, 19.765560150146484, -24.90596580505371, 28.06167221069336, -31.989341735839844, 15.6566743850708, 7.3419880867004395, -31.962505340576172, -7.964987754821777, -30.653657913208008, -10.980565071105957, -27.865558624267578, -4.699619770050049, 22.92879295349121, 15.14236831665039, -9.338122367858887, 43.2769775390625, 27.444581985473633, 4.514945983886719, -28.475238800048828, 38.90447235107422, 8.85079288482666, -11.70335865020752, 26.421245574951172, -27.91351890563965, -18.289588928222656, 37.551490783691406, 30.41176986694336, -54.2485237121582, 31.378576278686523, 5.528908729553223, -69.59786987304688, -4.569555282592773, -21.31597900390625, -5.143688201904297, -86.07691955566406, -8.57520866394043, 17.787296295166016, -16.853057861328125, -5.2028985023498535, -9.63221263885498, 38.08190155029297, 36.99846649169922, -25.28424835205078, -13.285989761352539, -48.33852005004883, 31.112865447998047, -44.3946533203125, 9.970412254333496, -15.154747009277344, -0.4028036296367645, 40.37089157104492, 26.077573776245117, -13.702244758605957, -7.152800559997559, 6.910159111022949, -17.8187313079834, -50.92169189453125, -10.05099868774414, 28.911861419677734, 49.09067916870117, -25.015274047851562, 1.340631365776062, -7.230962753295898, -0.6683765053749084, -10.869994163513184, 14.262197494506836, 1.1464858055114746, 38.56285095214844, 2.050617218017578, 0.7954674959182739, 33.17762756347656, -56.62348556518555, 6.914578914642334, -0.04824502393603325, 33.0693359375, 2.8861887454986572, -16.831472396850586, 26.289535522460938, 9.506207466125488, -21.119869232177734, 3.9505882263183594, 59.3214111328125, -58.5700569152832, 45.386653900146484, -31.035808563232422, -24.57857322692871, -5.746735095977783, 42.817787170410156, 12.26512622833252, 19.745805740356445, -3.6465022563934326, 23.84554672241211, 27.960458755493164, -14.310775756835938, 17.668926239013672, 8.933650970458984, 37.19678497314453, -3.847304344177246, 39.5654296875, 19.87370491027832, 8.480334281921387, -14.418214797973633, 26.72032356262207, 25.221467971801758, 22.06952476501465, 7.7320475578308105, 1.7708449363708496, -2.6308112144470215, -28.349328994750977, 17.263240814208984, 4.390636920928955, 55.362125396728516, 0.5319522619247437, -18.769981384277344, 54.69068145751953, 5.1343607902526855, 1.200003743171692, 4.963387489318848, -40.47052764892578, 28.719402313232422, -49.34865951538086, -9.7426176071167, 15.958755493164062, 16.277196884155273, 13.521467208862305, 41.938148498535156, -38.74824905395508, 9.155105590820312, 8.663743019104004, -10.994007110595703, -12.90999698638916, 2.4918668270111084, -21.27288055419922, 20.742460250854492, -41.125247955322266, -17.11224365234375, -6.308166980743408, -44.270137786865234, -25.157550811767578, 48.30562973022461, 44.657615661621094, 19.605316162109375, 4.593327045440674, 35.216514587402344, 32.48192596435547, -3.224968433380127, -9.993300437927246, 11.5220947265625, 54.05345916748047, -25.108768463134766, -12.3240327835083, -11.27496337890625, -21.556093215942383, 0.18984445929527283, -31.293655395507812, 18.825815200805664, -11.959418296813965, -5.957667350769043, -48.20170593261719, 33.33869934082031, 44.44186782836914, 31.014759063720703, -30.141056060791016, -29.184364318847656, 17.02843475341797, 5.545510292053223, 34.427486419677734, 32.36796188354492, 2.4024617671966553, 5.51232385635376, -23.3773250579834, 14.915399551391602, 8.492288589477539, 28.612119674682617, -4.501531600952148, 78.48421478271484, 17.655759811401367, 26.51357078552246, -13.234189987182617, 72.67910766601562, 18.69316864013672, -31.12441062927246, 6.3522138595581055, -23.076698303222656, 14.624905586242676, -1.8001227378845215, 6.056543350219727, 9.329438209533691, -10.843193054199219, -2.987661600112915, 29.773700714111328, -3.800649404525757, 31.39650535583496, 46.070465087890625, 32.486053466796875, 15.57070255279541, 8.179056167602539, 26.4450626373291, -24.310558319091797, -20.282503128051758, -7.315276145935059, -22.079055786132812, 56.609375, 5.099260330200195, -7.031103610992432, -17.238515853881836, -50.96858596801758, 16.50727653503418, 21.7499942779541, 10.285871505737305, -3.7567551136016846, 52.93268966674805, 54.134246826171875, 5.284161567687988, 41.99217987060547, 14.061367988586426, 1.967036485671997, -11.652920722961426, 0.024549230933189392, -27.387706756591797, 34.05699920654297, 26.950510025024414, -30.653644561767578, -19.03251838684082, 31.670930862426758, -75.02020263671875, -6.897994041442871, 44.97314453125, -2.48565673828125, 45.58378219604492, -25.886213302612305, -32.66268539428711, 9.286930084228516, 61.05350875854492, -13.308942794799805, 41.125892639160156, -19.0978946685791, -2.6811954975128174, -59.91340637207031, -19.430593490600586, -6.027818202972412, -2.4138195514678955, -19.263341903686523, 19.988187789916992, 27.011781692504883, 6.968864917755127, -36.63487243652344, 21.4688777923584, -7.123601913452148, -14.52049732208252, 26.926740646362305, -16.251768112182617, 24.178905487060547, 16.87386703491211, -16.732139587402344, -17.91371726989746, 2.346390724182129, 19.275959014892578, 32.68265914916992, 59.71156311035156, -13.329545021057129, -12.742956161499023, 19.67696189880371, -23.699350357055664, -47.33895492553711, -45.73793029785156, 37.88059997558594, -14.296197891235352, -42.61714172363281, 102.06410217285156, 1.0729507207870483, -35.571815490722656, -8.133292198181152, 8.263873100280762, 3.6646039485931396, -1.8390449285507202, -0.04929620027542114, 4.348363876342773, -22.762842178344727, -98.12261962890625, 9.458719253540039, 44.23999786376953, -1.0282903909683228, -27.268051147460938, 24.03271484375, 12.249495506286621, -2.9597272872924805, -12.371580123901367, -9.37411117553711, -4.432198524475098, -40.88673782348633, -38.19706726074219, 9.244202613830566, 14.862076759338379, 36.68537902832031, -27.038143157958984, -20.32762336730957, 5.103484630584717, -44.930694580078125, 17.69642448425293, -14.172229766845703, 19.84258460998535, -43.60179138183594, -11.49206829071045, 36.601680755615234, 14.715420722961426, -23.810678482055664, -9.746384620666504, -5.461138725280762, -34.46369171142578, -14.35340404510498, 49.76232147216797, 64.47274017333984, 9.616022109985352, 52.16752243041992, -27.242000579833984, -20.086227416992188, -7.159168243408203, -4.980372428894043, 2.1249184608459473, -10.082704544067383, -84.28511810302734, -13.842947959899902, 11.398229598999023, -11.404948234558105, 69.5364990234375, 21.503480911254883, 63.49667739868164, 1.0568312406539917, 12.163188934326172, 24.53404426574707, 12.606063842773438, 11.626431465148926, 10.16648006439209, -5.737936019897461, 7.304081439971924, 14.32283878326416, -17.0190486907959, 38.06490707397461, 3.102377414703369, 42.098751068115234, 8.382399559020996, 31.086143493652344, 5.984721660614014, 50.67801284790039, 42.496273040771484, 17.39759635925293, 26.262901306152344, 40.00893020629883, -5.145237445831299, 3.098717212677002, 13.931673049926758, -48.113094329833984, 16.864051818847656, -37.383235931396484, -40.435211181640625, -6.900267124176025, 35.245765686035156, -11.347481727600098, 24.665739059448242, 14.751943588256836, 22.325172424316406, -29.613779067993164, 54.87192916870117, -6.01123571395874, 50.18360900878906, 39.73807907104492, -50.052154541015625, 47.52076721191406, 18.29393196105957, -8.158896446228027, -63.70966720581055, 6.166877269744873, 51.70640563964844, 53.504337310791016, 5.244927406311035, 10.873239517211914, -21.7977237701416, -25.781606674194336, 11.848806381225586, -1.0313202142715454, -16.91486167907715, 2.379748821258545, -55.446998596191406, 22.186851501464844, -51.34104537963867, -24.773303985595703, 1.7409523725509644, -17.872413635253906, -8.586214065551758, 25.452789306640625, 5.644705295562744, -13.904263496398926, 42.85604476928711, -8.744302749633789, -3.674839973449707, 32.20750427246094, -16.583431243896484, 24.653959274291992, -4.486089706420898, -23.77513885498047, -42.07255172729492, -52.24727249145508, -33.21446228027344, 19.355688095092773, 1.7119537591934204, 5.908285617828369, 4.1677470207214355, -13.683389663696289, 31.446962356567383, -9.008736610412598, -56.6923942565918, 14.928624153137207, 47.81649398803711, -15.33080768585205, 1.5933382511138916, 14.28660774230957, -30.105154037475586, -45.391624450683594, 2.905121088027954, 35.755924224853516, 30.974811553955078, 4.489275932312012, -10.642877578735352, 10.964673042297363, 19.90488624572754, 25.136816024780273, 57.76422119140625, 4.809375762939453, -24.28118324279785, -0.9406100511550903, -38.068031311035156, -13.575236320495605, 0.7446022033691406, 3.0101239681243896, -11.317319869995117, 63.609344482421875, -21.17780113220215, 8.94409465789795, 99.03654479980469, -10.420662879943848, -28.962587356567383, -26.253862380981445, -16.572795867919922, 41.153541564941406, 3.3233518600463867, -32.449951171875, -0.4440408945083618, -19.985109329223633, -84.7539291381836, -45.000240325927734, -14.705400466918945, -41.9062385559082, 30.491024017333984, 17.017494201660156, -31.062702178955078, 48.8213996887207, -1.9623289108276367, 1.755064845085144, 23.696605682373047, 15.442391395568848, 37.10417175292969, -9.238578796386719, -7.2141265869140625, -16.93785858154297, 33.99174118041992, 38.859256744384766, -26.440357208251953, -44.24921798706055, -12.85321044921875, -54.80942916870117, -23.25657081604004, -23.631244659423828, -7.287728309631348, 39.063568115234375, 15.184598922729492, -57.085792541503906, -26.120777130126953, -34.520328521728516, -23.444978713989258, -26.28547477722168, -2.9946184158325195, 16.595895767211914, 3.736111640930176, 71.64368438720703, -8.968542098999023, 43.80906677246094, 3.252549171447754, 18.44069480895996, -46.5180549621582, 23.80171012878418, -28.342573165893555, -6.628271102905273, 10.73421859741211, 15.331392288208008, 28.433055877685547, 10.139830589294434, -39.2547607421875, -37.04692077636719, -17.968366622924805, 3.876765251159668, -27.576475143432617, -9.18452262878418, -0.6529744267463684, -6.667973041534424, 22.69044303894043, -28.99694061279297, -25.446260452270508, -32.544349670410156, 12.09292984008789, -17.266178131103516, -22.593637466430664, 19.97658348083496, 27.9096622467041, -29.031208038330078, 24.410478591918945, -35.31647872924805, 28.054685592651367, -18.784107208251953, 13.697071075439453, 9.66704273223877, 20.73883628845215, -2.930856466293335, -19.650938034057617, -3.5611233711242676, 30.69577980041504, 1.6669715642929077, 31.52095603942871, 76.59243774414062, 14.600090026855469, 35.159278869628906, -4.156309604644775, -6.738130569458008, -9.243852615356445, 17.246681213378906, 77.4038314819336, 4.735926628112793, -14.178235054016113, -13.964550971984863, -34.577415466308594, 24.710269927978516, -8.692858695983887, -60.83237838745117, 12.49798583984375, -60.839439392089844, 14.960039138793945, 30.556480407714844, 27.029155731201172, -21.299785614013672, 45.02888488769531, 46.7534065246582, 30.06814193725586, 8.321739196777344, -18.482542037963867, 47.72196960449219, 1.0020849704742432, -65.10485076904297, 6.6596879959106445, 41.64046096801758, -2.002962827682495, 15.178152084350586, -28.97414207458496, -4.386120319366455, 25.374589920043945, 1.3353029489517212, 27.19801902770996, 8.75566291809082, 42.22677230834961, -48.42531967163086, -11.533763885498047, -37.729522705078125, 1.6191741228103638, -53.83964157104492, -29.172000885009766, -40.56714630126953, -38.26493453979492, 27.264270782470703, -43.87825393676758, -24.91741371154785, -27.56509017944336, 17.48506736755371, -28.01688575744629, 10.691283226013184, -20.593700408935547, 18.69535255432129, 21.61077880859375, 11.846423149108887, -47.34774398803711, 64.50997161865234, -55.094757080078125, 31.835350036621094, -20.259475708007812, 6.608492851257324, -63.29274368286133, 19.161222457885742, -29.57271957397461, 11.332184791564941, -3.0966217517852783, -15.1509370803833, -27.93347930908203, 9.748757362365723, 28.27199935913086, 28.4459285736084, 23.126081466674805, -32.725067138671875, 1.372125267982483, -3.917945146560669, -2.738480806350708, -29.875757217407227, 17.329362869262695, -2.560178279876709, 30.942781448364258, -15.693937301635742, -38.928062438964844, -5.294456481933594, 29.75983238220215, 35.18595504760742, 27.355222702026367, -43.228309631347656, 26.84548568725586, 17.853784561157227, -44.811370849609375, 65.91549682617188, 12.048189163208008, 24.04114532470703, -48.25260543823242, -54.30796432495117, 29.956762313842773, -58.46282958984375, 48.432647705078125, 6.8233184814453125, -11.017276763916016, -14.1925630569458, -71.89620971679688, 17.241275787353516, 12.181499481201172, -18.492948532104492, -20.25888442993164, 14.440821647644043, 25.032377243041992, 13.605154991149902, -18.116857528686523, -8.226845741271973, -46.48469543457031, 19.108501434326172, 24.678939819335938, 7.178760051727295, -14.87047290802002, 19.3457088470459, 0.22718724608421326, -13.882485389709473, 10.977302551269531, -0.5504659414291382, -55.79393768310547, 13.688142776489258, 26.044158935546875, 63.5677375793457, -45.2220458984375, -6.100398063659668, 60.942012786865234, 38.93785858154297, 37.65810012817383, 26.365276336669922, -6.518200397491455, -33.14289093017578, 30.51100730895996, 55.61592483520508, -47.480037689208984, -17.286821365356445, -18.850738525390625, 35.215457916259766, 21.980083465576172, -2.518650770187378, -53.18293762207031, 16.616384506225586, -0.2879989743232727, 2.2087864875793457, 3.117295265197754, -24.934499740600586, 11.691474914550781, -42.06991958618164, -38.15532684326172, -6.7960333824157715, -23.634334564208984, -5.6376543045043945, -11.612517356872559, -54.24470138549805, 8.343408584594727, 12.589298248291016, -12.497200965881348, -17.50018882751465, 4.876561641693115, -73.52771759033203, -29.425086975097656, 2.418919086456299, 21.120100021362305, 9.54156494140625, -15.793991088867188, -2.676894426345825, -7.431455612182617, -19.541820526123047, -51.88024139404297, -15.693763732910156, 20.872034072875977, 55.74502944946289, 38.87662124633789, 33.54735565185547, 20.448678970336914, 35.65983963012695, -23.34425163269043, -26.140701293945312, -28.505064010620117, -15.707808494567871, 7.368330955505371, 47.09062576293945, -7.5429840087890625, 14.308308601379395, 21.482135772705078, 17.664535522460938, -19.622432708740234, -15.689083099365234, -6.202478408813477, 42.51503372192383, 21.24537467956543, 31.963359832763672, -62.36073684692383, -29.460956573486328, -4.9858479499816895, -27.6539306640625, 53.855716705322266, 0.25477826595306396, 30.81653594970703, 34.54120635986328, 21.151321411132812, -5.463681221008301, 67.53611755371094, 31.905261993408203, -22.435890197753906, 13.523933410644531, -9.593497276306152, -6.832605361938477, -29.62872314453125, 10.610304832458496, 29.072410583496094, 28.333988189697266, 26.87782096862793, 14.663541793823242, -6.737188816070557, -7.483669281005859, 3.672661304473877, 36.07861328125, 0.2649018168449402, -42.017032623291016, -6.563279628753662, 4.463700294494629, 10.939763069152832, -19.782419204711914, 69.63900756835938, 54.10222625732422, 65.89645385742188, 91.3550033569336, -18.025022506713867, 23.485435485839844, 16.669639587402344, 13.682734489440918, -32.26948547363281, 19.25843620300293, 1.7177857160568237, -4.646884918212891, 35.04822540283203, -13.510185241699219, -26.779321670532227, -23.98956871032715, -8.379707336425781, 0.43734532594680786, -53.06127166748047, 9.8753023147583, -24.605606079101562, 7.624982833862305, 33.19179916381836, -14.581686973571777, -33.71221160888672, 22.530622482299805, 25.35982322692871, -43.28282165527344, 3.467043876647949, -58.89766311645508, 2.30248761177063, 6.500884056091309, -1.049246907234192, 24.886632919311523, 10.256598472595215, 60.84427261352539, -13.899898529052734, 33.4075927734375, -2.3996524810791016, -19.812559127807617, -46.91789245605469, -46.28062438964844, -34.4829216003418, 6.7188897132873535, 36.974517822265625, 4.706833839416504, 76.25454711914062, 1.3737353086471558, 34.54446029663086, 45.324527740478516, -13.374826431274414, -27.315244674682617, 7.726925849914551, 24.17936897277832, -24.13657569885254, 10.411109924316406, -53.17747116088867, -11.353596687316895, -2.903979539871216, -77.4904556274414, 7.308056831359863, -41.23924255371094, -36.86980438232422, -7.351283073425293, 2.1922972202301025, -10.523730278015137, 32.9118766784668, -26.297008514404297, -32.90644836425781, -4.471453666687012, -50.7805290222168, -4.227956295013428, -0.5300827026367188, 9.761991500854492, -52.3769416809082, -35.72332763671875, 12.300555229187012, 7.12510871887207, -57.001220703125, -16.874279022216797, 5.277787685394287, 16.57723617553711, 35.65201950073242, 11.41243839263916, -13.039247512817383, 17.829872131347656, 8.89335823059082, -12.477011680603027, 64.375244140625, 12.215245246887207, 3.837101697921753, -2.786269187927246, 21.572113037109375, 0.24273844063282013, 76.98454284667969, 11.010129928588867, -40.107357025146484, 49.008243560791016, 2.3287484645843506, 68.66305541992188, 26.792724609375, 23.28106117248535, 1.2113382816314697, 41.42229080200195, -10.1233491897583, -13.919878959655762, 48.62812042236328, 35.86163330078125, 4.991787910461426, 17.11932945251465, -45.91191864013672, 25.194002151489258, -5.853044509887695, 70.8243637084961, 11.867254257202148, 12.559178352355957, -5.944453239440918, -17.43225860595703, -6.251982688903809, 31.31412124633789, 3.9848833084106445, 16.316162109375, 18.723432540893555, 8.103519439697266, 2.3450658321380615, 65.28032684326172, 7.520281791687012, 29.60929298400879, -18.126232147216797, 14.572097778320312, -15.735503196716309, 37.64985656738281, -5.408425807952881, 54.72592544555664, 8.689864158630371, -12.040839195251465, 14.976865768432617, -31.722719192504883, 22.12369155883789, 28.40155601501465, -30.163610458374023, 44.274559020996094, -1.671078085899353, -12.22026252746582, 8.730952262878418, -42.48230743408203, 0.011189122684299946, -1.050120234489441, 5.5770673751831055, 42.334571838378906, 2.8149008750915527, -3.0674569606781006, 20.87516212463379, 25.789609909057617, -2.7757577896118164, -11.908226013183594, 3.077131509780884, -37.21902084350586, 7.298574924468994, -51.16777420043945, 12.353686332702637, -10.949938774108887, -1.3874108791351318, 4.1320624351501465, -12.127649307250977, 7.888431072235107, -78.22967529296875, 66.3698959350586, -44.89378356933594, 36.38356399536133, 35.62407684326172, 14.175019264221191, -1.9821066856384277, -3.7824461460113525, -8.381593704223633, -2.4697508811950684, 21.165550231933594, 8.081953048706055, -22.758792877197266, 16.710336685180664, 36.022708892822266, -24.356632232666016, -7.064470291137695, -7.571728706359863, -41.87134552001953, 56.829647064208984, -51.70408630371094, 50.05134201049805, 5.300471782684326, -49.13884353637695, 34.78376770019531, 13.576011657714844, 56.63083267211914, 14.421927452087402, -5.669170379638672, -14.356194496154785, -17.570301055908203, -30.13611602783203, 9.252511978149414, -50.81000518798828, 90.48692321777344, -39.757816314697266, 46.062503814697266, -37.72857666015625, -37.85926055908203, 15.766627311706543, 2.9946224689483643, 2.7082200050354004, 23.32819366455078, 49.345306396484375, 6.537773132324219, 23.58221435546875, 18.628618240356445, -9.807987213134766, 19.368165969848633, -8.206762313842773, -17.458288192749023, -32.26224136352539, 16.6428279876709, -48.73834991455078, 29.270978927612305, -13.922524452209473, 7.406257629394531, -9.105137825012207, 29.590778350830078, -21.022722244262695, 11.427546501159668, -52.17433547973633, 15.515928268432617, -24.56905174255371, -23.567737579345703, -7.942389488220215, -18.38550567626953, 48.949771881103516, 42.32098388671875, 34.166969299316406, -20.436294555664062, -21.56955337524414, -3.925853729248047, 22.096540451049805, -65.93781280517578, -18.524385452270508, -33.99195861816406, -25.752880096435547, -37.97365951538086, -44.018558502197266, -5.222512722015381, -35.022216796875, -13.860453605651855, -13.560090065002441, 9.899679183959961, -59.29313659667969, 23.535470962524414, 7.475377559661865, -9.619229316711426, 31.27646255493164, 29.574247360229492, 29.895896911621094, 14.385275840759277, -40.66368865966797, -26.527389526367188, -26.635807037353516, 11.022528648376465, -42.58715057373047, -12.96500301361084, 41.574222564697266, 19.082298278808594, 49.17667770385742, -13.53003978729248, -56.29536819458008, 67.10809326171875, -11.339591979980469, 18.51578140258789, -17.24428367614746, 2.2014682292938232, 46.625484466552734, -54.32915115356445, 55.84291458129883, 1.932074785232544, 7.790611267089844, 25.89543342590332, 33.727699279785156, -10.13841438293457, -15.628946304321289, 3.992655038833618, -12.775141716003418, -9.226707458496094, -27.653902053833008, 19.46804428100586, 48.44670486450195, -6.382456302642822, -41.37458801269531, -44.72235870361328, 2.8704400062561035, -55.999488830566406, -14.769932746887207, -0.6699432730674744, 15.900172233581543, 3.334766387939453, -14.196776390075684, 24.170503616333008, 42.124298095703125, -15.536956787109375, 32.40452194213867, -7.665637016296387, 22.916725158691406, 34.89432144165039, 17.631147384643555, -20.737361907958984, 15.458818435668945, 12.38857364654541, 5.00823450088501, -1.5090110301971436, -26.2125186920166, -41.64330291748047, -19.793071746826172, -43.53937911987305, 9.502748489379883, 27.45736312866211, -5.834959983825684, -27.64753532409668, 7.423241138458252, 11.385232925415039, -12.17889404296875, -79.05209350585938, 13.052007675170898, -15.790146827697754, 19.59654998779297, 18.10741424560547, -2.0976290702819824, 1.6980295181274414, 32.18294906616211, -53.94963455200195, 0.6495838165283203, 7.637831687927246, 25.838573455810547, -19.29390525817871, -30.412227630615234, 36.37141036987305, 48.35820388793945, -3.749764919281006, 4.9385151863098145, -33.2983512878418, -10.700818061828613, -0.006362095475196838, -0.0724041759967804, 47.78558349609375, -21.242374420166016, -18.90640640258789, -46.67879867553711, 34.30427169799805, 37.225502014160156, -5.530213832855225, 50.66512680053711, 10.220785140991211, 48.521121978759766, 11.732954025268555, 33.77614212036133, -32.66011428833008, -34.998897552490234, 23.39978790283203, 6.138701438903809, -15.292988777160645, 10.445762634277344, 2.368269920349121, 10.181440353393555, -9.049019813537598, 1.3854466676712036, 9.164013862609863, 52.615169525146484, 12.24495792388916, 14.186494827270508, -50.543819427490234, 10.71202564239502, -40.687713623046875, 62.17374038696289, -0.6569196581840515, 25.087032318115234, 16.611984252929688, 11.258719444274902, 0.37885135412216187, 7.591438293457031, 13.445228576660156, 5.428708076477051, -6.8431878089904785, -13.729455947875977, 15.07315444946289, 4.973641872406006, 4.851560115814209, 20.693662643432617, -5.393043041229248, -32.30401611328125, -4.356607913970947, 17.058658599853516, -17.899690628051758, -42.460025787353516, 23.60069465637207, 39.78197479248047, -8.881787300109863, 9.841617584228516, -68.89161682128906, -40.42146301269531, 5.529584884643555, 11.20844554901123, -53.9875373840332, -5.373883247375488, -14.808012008666992, -53.2293586730957, 25.128219604492188, -12.277740478515625, 21.140567779541016, 8.427831649780273, -24.9666690826416, 5.79069185256958, -20.031831741333008, 1.3623825311660767, 5.101415157318115, 41.5548210144043, 10.070428848266602, 25.045162200927734, 44.93244934082031, 17.857891082763672, -2.785921573638916, -24.659975051879883, 58.49961471557617, -12.903326034545898, 50.0451774597168, -30.01763153076172, -20.80520248413086, 11.970876693725586, -37.96670150756836, 22.19707679748535, -81.71388244628906, -40.597198486328125, 46.74246597290039, 3.16030216217041, 1.0504735708236694, 35.10298538208008, -23.041561126708984, -17.606536865234375, 24.891218185424805, -33.89035415649414, 18.76692771911621, 17.37970542907715, 63.727569580078125, 36.259159088134766, -4.274899959564209, 2.8868954181671143, -0.7979453206062317, -76.02093505859375, 26.441917419433594, 33.62907791137695, 33.439144134521484, -39.72303009033203, 38.522579193115234, -59.84156799316406, 3.164475440979004, -11.478687286376953, 9.695871353149414, 40.087894439697266, 2.0485055446624756, 56.14576721191406, 36.56670379638672, -0.40500420331954956, 15.970626831054688, 27.65863800048828, -24.109546661376953, -31.710601806640625, -25.28182601928711, -3.9168522357940674, 38.21214294433594, 44.7646484375, 28.138179779052734, -9.689903259277344, 10.887308120727539, 34.427955627441406, -8.519975662231445, -42.70232009887695, -31.691600799560547, -9.126702308654785, -26.779287338256836, -21.887361526489258, 46.54792785644531, 20.113643646240234, 13.312163352966309, -37.19847106933594, -31.75016975402832, -32.910133361816406, 85.47048950195312, -19.70972442626953, -15.430888175964355, -17.84400177001953, -16.457834243774414, 6.436267375946045, -60.35824203491211, 1.648014783859253, 4.325289249420166, 36.635284423828125, 31.24262046813965, -88.6338119506836, -18.197750091552734, 6.533181667327881, -41.163082122802734, 15.161903381347656, -0.3962228298187256, 0.5853140354156494, 25.591089248657227, 16.14328384399414, -10.450044631958008, 25.33820915222168, 5.115805625915527, 15.531116485595703, -17.90272331237793, 28.832454681396484, 1.4132072925567627, -2.2374513149261475, -59.75615310668945, -2.9409444332122803, -38.42957305908203, -13.326066970825195, 26.834205627441406, 19.467605590820312, -1.1405903100967407, -35.9764518737793, 13.429296493530273, 8.713298797607422, -5.379120826721191, 71.14258575439453, -13.273122787475586, -15.392786026000977, 0.5929548740386963, -12.634313583374023, 5.854814529418945, 38.76839828491211, 0.2720901668071747, -20.47412109375, 39.577056884765625, -14.55383586883545, -24.50089454650879, -12.801568031311035, 34.90719223022461, -38.58053207397461, -32.33976745605469, -22.176429748535156, -36.72764587402344, -27.570024490356445, -49.8293342590332, -3.017896890640259, -56.8199462890625, 2.740924596786499, -6.2770609855651855, 3.6894352436065674, 24.230030059814453, -9.647662162780762, 19.30227279663086, -14.889629364013672, 19.7273006439209, -3.9805259704589844, -14.759723663330078, -31.35524559020996, -52.891998291015625, -54.594581604003906, 74.6286849975586, 28.278860092163086, 9.371773719787598, 18.868471145629883, 9.557074546813965, -32.63608932495117, -36.058101654052734, 34.201637268066406, 29.369815826416016, -29.213903427124023, 18.259429931640625, 18.215913772583008, -42.327491760253906, -8.572492599487305, -23.975318908691406, -37.366065979003906, 25.112668991088867, -35.36167526245117, -11.720707893371582, 8.348542213439941, 28.341020584106445, -14.777021408081055, 22.411104202270508, 25.78636360168457, 28.13433265686035, -32.942806243896484, 36.968666076660156, 69.00775146484375, 53.41252899169922, 8.196287155151367, -5.794607639312744, -25.363571166992188, 7.843386173248291, 59.26428985595703, -17.60036277770996, -36.64105224609375, -15.271357536315918, 7.495070934295654, 37.807613372802734, -24.076690673828125, 54.2976188659668, -52.88100051879883, -7.927691459655762, -30.79447364807129, -28.986610412597656, -72.5417709350586, -29.384450912475586, 28.7558536529541, -30.282480239868164, -22.61479949951172, -33.451507568359375, -1.5934643745422363, -58.619056701660156, -3.749250650405884, -7.357611656188965, -22.57990837097168, -34.17040252685547, 46.66407775878906, 24.77246856689453, 52.11726379394531, 36.325260162353516, -46.122314453125, 13.657066345214844, 7.85714864730835, -18.314058303833008, 40.44945526123047, -11.876001358032227, -6.312460899353027, 12.874943733215332, 12.853225708007812, 55.397159576416016, 48.767601013183594, 5.642821311950684, 14.019908905029297, -12.86915111541748, -21.32269287109375, -16.233402252197266, 20.321243286132812, 7.596185684204102, 30.677278518676758, 15.311052322387695, -36.911094665527344, -27.58540153503418, 3.1787352561950684, 1.6147571802139282, 0.2716851532459259, -14.029683113098145, -12.841498374938965, -9.73213005065918, 4.088566303253174, -5.798211574554443, -0.6361499428749084, -26.592529296875, 8.013304710388184, 59.65293884277344, -20.581174850463867, -5.496092796325684, -10.872490882873535, -5.624125957489014, 46.923282623291016, -14.781067848205566, -21.073869705200195, -67.26058197021484, -56.070587158203125, -8.164522171020508, -21.827463150024414, -21.647573471069336, 12.852422714233398, 17.486373901367188, -62.52634048461914, 13.42170524597168, 70.66154479980469, -24.37851905822754, -70.7774658203125, 15.236059188842773, -5.330511093139648, -40.05037307739258, 15.315713882446289, 8.786690711975098, 8.455080032348633, -1.8492459058761597, 28.70463752746582, -45.374961853027344, -36.78888702392578, -11.24329948425293, 9.549525260925293, 17.739320755004883, 22.03597068786621, -27.859907150268555, 25.629079818725586, -15.953842163085938, -45.15966033935547, 72.98822021484375, -26.933025360107422, 16.721193313598633, -5.998661994934082, 20.168701171875, -9.0045747756958, -17.36155891418457, -9.15810489654541, 20.83304214477539, -96.98133850097656, -47.713985443115234, -32.74788284301758, -34.610843658447266, -15.909061431884766, 52.8646354675293, -14.885300636291504, 22.358234405517578, -37.222434997558594, -47.376441955566406, -44.634769439697266, -22.17599868774414, 19.908390045166016, 9.107930183410645, -3.499241352081299, -21.064952850341797, 1.9165395498275757, -6.944681644439697, 19.042423248291016, 65.42768096923828, -27.367366790771484, -37.519683837890625, 11.44307804107666, -19.085256576538086, -58.13288116455078, 26.94818878173828, -46.07133865356445, 13.26341438293457, 37.64781951904297, -41.614994049072266, 4.326173782348633, 28.233713150024414, -12.663689613342285, -52.15212631225586, -15.359210014343262, 13.660982131958008, -32.20132064819336, -6.5419840812683105, -6.262741565704346, -6.492545127868652, 3.5915772914886475, 0.11206409335136414, 31.48188018798828, 13.276150703430176, -18.548179626464844, -50.928733825683594, -3.8279643058776855, 16.6052303314209, -7.371554374694824, 38.318359375, 48.22757339477539, -1.5814788341522217, 20.065765380859375, -44.290836334228516, 17.991342544555664, 22.690698623657227, -8.493626594543457, -21.97936248779297, 8.950235366821289, -50.3083381652832, -46.015380859375, -26.495365142822266, 44.62783432006836, -20.346776962280273, -14.903106689453125, -73.81900024414062, 12.156342506408691, 31.392480850219727, 17.109928131103516, 14.644987106323242, -11.420191764831543, 2.6475396156311035, 0.7989762425422668, 7.505530834197998, -17.874542236328125, 25.22820472717285, -32.456207275390625, -37.862709045410156, -0.2296450436115265, -8.434158325195312, -57.599517822265625, 26.90921974182129, 20.417713165283203, 3.510944366455078, -21.648086547851562, -7.113305568695068, -9.167455673217773, 24.897602081298828, 68.04243469238281, 43.37053680419922, -39.82879638671875, 26.616411209106445, -20.106605529785156, -6.369114875793457, 0.37030643224716187, -26.92683982849121, 8.119016647338867, 21.113985061645508, -26.161901473999023, -66.85865020751953, -32.86803436279297, -19.387510299682617, -12.171186447143555, -5.587774276733398, -1.3494349718093872, -55.807254791259766, -54.03963851928711, 26.86988639831543, -10.007535934448242, -4.613761901855469, -63.30119705200195, -7.670837879180908, 9.022736549377441, -41.81597900390625, -12.850799560546875, 27.11520004272461, 14.941147804260254, 38.69037628173828, -30.683780670166016, 3.2357523441314697, 3.6458444595336914, -3.643247127532959, -39.825008392333984, -0.22836679220199585, 13.67969799041748, 46.445491790771484, 12.195993423461914, 11.258652687072754, 13.9387788772583, -38.79971694946289, -22.97256088256836, -3.588203191757202, -3.16232967376709, 27.019412994384766, 38.82201385498047, 22.449020385742188, -30.396995544433594, 20.447633743286133, 3.1893184185028076, 63.334136962890625, -28.758941650390625, -5.372941970825195, -5.995147228240967, 10.928678512573242, 93.6626205444336, 26.260560989379883, -20.65852165222168, -28.04300880432129, 3.192190647125244, -10.762490272521973, 24.78803253173828, 2.33550763130188, -57.40217208862305, -14.785786628723145, -36.48309326171875, -39.00937271118164, -17.214080810546875, -34.14114761352539, 23.889692306518555, 15.244100570678711, 17.63263511657715, 13.18653678894043, -12.084920883178711, -57.45777893066406, 21.062856674194336, 32.045989990234375, 105.53152465820312, -8.700447082519531, -38.19966125488281, -7.017233848571777, 10.600347518920898, 12.378607749938965, 14.201959609985352, -79.18392181396484, 0.14701667428016663, 6.459560394287109, -52.02953338623047, 35.050933837890625, 16.134008407592773, -41.89128875732422, 12.588096618652344, 6.642579555511475, -2.2287886142730713, -13.343636512756348, 1.4889509677886963, 44.225975036621094]], "dtype": "float32"}, {"__nd__": [[1.7238765954971313, -0.5749472975730896, 0.5453272461891174, 0.25883248448371887, -0.6330980062484741, -0.012113062664866447, -0.7841876745223999, 0.79252690076828, 0.3295680284500122, 1.4873930215835571, -1.046878457069397, -0.9063586592674255, 0.3597361743450165, 1.8732467889785767, -0.6135297417640686, 0.8003865480422974, -0.9419698119163513, 0.17005260288715363, -0.7297120094299316, 1.1571961641311646, -0.2130574882030487, 2.2366113662719727, -0.22429011762142181, 0.05826137587428093, 0.6032696962356567, 0.737610936164856, 0.023678161203861237, 0.972224771976471, -0.302313894033432, 1.421953797340393, 1.8176722526550293, 0.44287043809890747, 0.837379515171051, 0.6210570931434631, 0.43464797735214233, 1.109706163406372, 0.9987831115722656, -0.42715421319007874, 0.8094421625137329, -0.5584933757781982, -0.3444012403488159, 1.2625985145568848, -0.22851574420928955, 3.3533153533935547, 0.25791388750076294, 1.8098292350769043, 1.9804058074951172, 1.349736213684082, -2.709836959838867, 0.3889657258987427, 0.7192005515098572, -0.5830459594726562, 0.2519100308418274, -1.095241904258728, -0.8272406458854675, -0.31125229597091675, 0.6086130142211914, 0.47367092967033386, 0.8925759792327881, 0.18591350317001343, -0.2076430469751358, 1.6804418563842773, -1.4046761989593506, -1.0824424028396606, 0.6327160000801086, -0.46594369411468506, 0.8577221035957336, -0.13132798671722412, -1.4972450733184814, -1.4005911350250244, 2.2518820762634277, -0.2817205786705017, -1.177436113357544, 1.524453043937683, -0.13769184052944183, 0.6757864952087402, 0.6153462529182434, 2.0712368488311768, -0.2564544975757599, 0.08550907671451569, -0.7323039174079895, 0.47230979800224304, 0.8004911541938782, -1.8660967350006104, -0.8133909106254578, -1.0809749364852905, -1.7695503234863281, -1.7899523973464966, -0.7493121027946472, -0.30875110626220703, -0.11729686707258224, -1.4338223934173584, 0.07552246749401093, -0.8476037383079529, -0.9551360011100769, 1.3502120971679688, 1.9274556636810303, -1.3906346559524536, -0.8113964796066284, 1.080471158027649, 0.020465871319174767, -0.3035675883293152, 1.8815854787826538, -1.4960227012634277, 1.7367695569992065, 0.007938672788441181, -0.643974781036377, -0.3392929434776306, -1.2227755784988403, 2.2029385566711426, -0.6631407737731934, -0.3651660978794098, 1.5942052602767944, 0.03301534429192543, 0.2396155595779419, 0.329111784696579, -1.0362803936004639, -1.0773673057556152, 0.36482805013656616, -0.1668112725019455, 0.2847420871257782, -1.3300225734710693, 0.5754587650299072, -1.1092865467071533, -0.05720173567533493, -0.2297602742910385, 0.04396497830748558, 0.16208651661872864, -2.2064573764801025, -0.11610688269138336, 0.4399507939815521, 0.8020865321159363, 0.9970673322677612, 0.20520739257335663, 1.0621064901351929, 0.41739532351493835, -0.36856403946876526, -0.4649695158004761, 1.6097007989883423, 0.7573699951171875, -0.6495270729064941, -1.7016282081604004, -1.0417953729629517, 1.4746088981628418, 2.3102824687957764, 0.8288062214851379, 1.8915098905563354, 0.9280052781105042, 0.3563125431537628, 0.10016395151615143, -0.047906044870615005, 0.37373873591423035, -0.5791043043136597, 0.4259275496006012, 0.6298475861549377, -1.0426931381225586, -1.597225308418274, 0.9712472558021545, 0.4146302342414856, 1.3378264904022217, -1.3152742385864258, -1.3468544483184814, 1.0622880458831787, -0.9511858224868774, -0.1644531935453415, -0.5017624497413635, 0.3467335104942322, 0.9883401393890381, 1.0531020164489746, -1.5232725143432617, 0.45906707644462585, -0.8516513705253601, -0.6183772087097168, 0.39298224449157715, 0.11479496210813522, -1.3253002166748047, -1.0399712324142456, -0.7844459414482117, -0.23616738617420197, -0.0004047650145366788, -0.04882798716425896, -1.078595757484436, -0.3170485496520996, -0.10611722618341446, -0.9357528686523438, -1.3505284786224365, 0.4537959396839142, 0.7893531322479248, 2.3267133235931396, -0.16909900307655334, -0.31686723232269287, -1.2961324453353882, -1.524755597114563, -0.4943084716796875, 0.49016857147216797, 0.9610323905944824, -1.2078334093093872, -0.3019724190235138, -0.0336875356733799, -0.6692810654640198, 0.41293013095855713, -0.9860551357269287, 1.524057149887085, -0.717690646648407, 1.2494288682937622, -1.0027121305465698, -0.4788777232170105, -1.3879913091659546, 1.3616833686828613, 1.5017811059951782, -1.0282444953918457, 0.03130227327346802, -0.38188910484313965, 0.2042652815580368, -0.2773222327232361, -1.0442720651626587, 0.5233882665634155, 0.32350045442581177, 0.366393119096756, 1.0218192338943481, -1.2651623487472534, -0.33023932576179504, -0.5998042225837708, 0.5953306555747986, -0.5388776659965515, 0.28512656688690186, -0.8406445384025574, -0.7807870507240295, -0.8802384734153748, -0.35578224062919617, 0.575055718421936, 1.0404648780822754, 0.15315639972686768, 0.04807685315608978, -1.0435998439788818, 0.17506656050682068, -0.5397564172744751, 0.7156438827514648, -0.13512474298477173, -1.3085339069366455, -1.513061761856079, 0.6951537728309631, 0.7677117586135864, -1.0103236436843872, -0.2800288498401642, 0.7484668493270874, -1.9375673532485962, 0.7523590326309204, -0.46485093235969543, 0.5548238158226013, 1.1345016956329346, -0.26261061429977417, -0.16846013069152832, -0.06197885051369667, -0.1706133931875229, -1.405509352684021, 0.27513378858566284, 1.2980308532714844, -0.7934295535087585, -0.14346933364868164, -1.0454634428024292, -0.16907694935798645, -0.4458962082862854, -0.5272789597511292, 0.7703887224197388, -0.14066295325756073, 1.247819185256958, -1.7671211957931519, 0.7917287945747375, -2.0696678161621094, -0.3436598479747772, 1.2196686267852783, -0.4718228280544281, 0.32516440749168396, -0.7148684859275818, 0.29109328985214233, 0.040557969361543655, -0.7439899444580078, 0.04164281487464905, -2.551452875137329, 0.5459828972816467, 0.2748074531555176, 0.25444355607032776, -0.968251645565033, -1.6733125448226929, -1.2775177955627441, 0.39444321393966675, 0.47577622532844543, 0.8319456577301025, -0.10742581635713577, -0.32557255029678345, -0.7484260201454163, 1.1606405973434448, -0.5209614038467407, 0.6627086997032166, 0.08465953916311264, -0.4296579360961914, -0.7447468638420105, -0.10649638622999191, 0.2660881578922272, -1.2707791328430176, -0.8931336998939514, 0.9020151495933533, -0.23728875815868378, 0.06644044816493988, -0.056941717863082886, 1.0887763500213623, -1.2388296127319336, -0.09512779116630554, 0.8395275473594666, 0.31228581070899963, -0.6638588309288025, -0.549494206905365, 0.21675236523151398, 0.27664440870285034, -0.4428626298904419, 0.9058521389961243, 1.1381176710128784, 0.5447078943252563, 0.04332819953560829, -0.00798873696476221, -0.7946362495422363, -0.44611090421676636, 0.17641980946063995, -1.6608707904815674, 0.20260609686374664, -0.767888069152832, -0.5266494750976562, 0.28598552942276, -0.07648008316755295, 1.0274358987808228, -1.0316261053085327, 0.18180467188358307, 1.3967983722686768, 0.9021516442298889, 0.4382324814796448, 0.6658279895782471, -0.2751321494579315, 0.19478794932365417, 0.7656012773513794, -0.6608772873878479, 0.716966986656189, 0.10690529644489288, -0.15237221121788025, 0.22276277840137482, 0.37575581669807434, -0.7988558411598206, -1.4028046131134033, 0.9279139041900635, -1.3631736040115356, -1.0152734518051147, 0.6633597016334534, -0.7425881624221802, 1.698797583580017, -1.1020286083221436, -0.06914452463388443, -1.2281413078308105, 1.0546923875808716, -0.41142019629478455, -0.043990399688482285, -0.8730295300483704, 1.1584539413452148, -1.2544825077056885, -0.8004186749458313, 0.4425507187843323, -1.597329020500183, -1.2520197629928589, -0.4223710298538208, -0.518329381942749, -1.4176609516143799, -1.146316409111023, -0.2736819088459015, 0.8841924667358398, 0.922213077545166, -0.7871072292327881, 0.4971476197242737, 1.139854073524475, 0.4472878575325012, 0.19973088800907135, 1.0123417377471924, -0.7237678170204163, -2.5820624828338623, -0.18985895812511444, 1.8758926391601562, 0.004295577295124531, 0.2782823145389557, -0.08690748363733292, 0.05077787861227989, 1.8880094289779663, 0.04339859262108803, 0.6204605102539062, -1.3153836727142334, -0.49255481362342834, 0.8446352481842041, 0.35378897190093994, 0.2695082128047943, 0.1347476691007614, -0.7308133840560913, 1.4624788761138916, 0.5206689834594727, -0.8837918639183044, 0.584409236907959, 0.3243039548397064, -0.025708960369229317, -0.6733405590057373, 0.41059818863868713, -0.4236368238925934, -0.30880358815193176, 0.8572942614555359, -0.8865442872047424, -1.598944067955017, -0.9626750349998474, -0.3715217709541321, 0.5526748895645142, -0.2848701775074005, 1.3374346494674683, 0.9192628264427185, 0.03836548328399658, -1.264035940170288, -0.4978761076927185, -0.3541312515735626, 0.5848968625068665, -0.12679322063922882, 0.02014928124845028, -0.4130190312862396, -1.2359282970428467, 1.045688271522522, 0.6595064997673035, 0.7094871401786804, -0.8833624720573425, 0.8754067420959473, -0.11184186488389969, -0.8040903210639954, 0.4582486152648926, -1.8516387939453125, 0.2457779496908188, 0.1817358285188675, -0.5299777984619141, 0.050292253494262695, -0.46829208731651306, 0.2511693835258484, -0.9157454371452332, 1.9794502258300781, 1.1345791816711426, 0.7358406782150269, -0.2892864942550659, -0.8397499322891235, 0.9446876049041748, 0.7169291377067566, 1.1587647199630737, -0.5607540011405945, -0.893575131893158, -0.5379098057746887, 0.3330017030239105, -0.10830461233854294, -0.3863506019115448, 0.26585379242897034, -1.0432484149932861, -0.7251838445663452, -0.18477630615234375, 0.7321299314498901, -0.9162538647651672, 1.9150656461715698, 1.6829086542129517, -1.0009853839874268, -0.4050012230873108, 0.07508186250925064, 0.7896119952201843, 1.5240517854690552, 1.1901731491088867, 1.4947166442871094, 1.2096835374832153, 0.11855155229568481, -0.3904065787792206, -0.4181225299835205, -0.46553879976272583, -0.0927458181977272, -0.07967172563076019, -0.5719033479690552, 0.29780226945877075, -0.7178995609283447, 1.1037999391555786, -0.7955867052078247, -1.3642141819000244, -0.3516496419906616, -0.12097665667533875, 0.8701377511024475, 0.22200776636600494, -0.596442699432373, -0.7315966486930847, 1.3444331884384155, -1.2797476053237915, -0.4929535984992981, -0.07152952998876572, 0.04896144941449165, 1.6260778903961182, 1.1749095916748047, -0.6297655701637268, -1.5971314907073975, 0.10747021436691284, 2.3347482681274414, -0.02225309982895851, -1.4298630952835083, 0.07521161437034607, -0.43254372477531433, 1.1512248516082764, 0.06646114587783813, -0.11359473317861557, 1.081313967704773, 0.5568195581436157, -1.056152582168579, 0.4899657070636749, 0.9546833038330078, 1.0908408164978027, -0.7098596096038818, -0.9241841435432434, -0.9220011830329895, 1.1127312183380127, 0.08349219709634781, -1.2707325220108032, 0.39023616909980774, 2.4342823028564453, 0.9079708456993103, -0.720514714717865, 0.23134110867977142, -0.9903495907783508, 0.27312958240509033, -1.2044578790664673, 0.30394819378852844, -1.230650544166565, -1.6722462177276611, 0.8007797598838806, -0.304517924785614, 0.026714017614722252, -0.881532609462738, -0.5986961722373962, -0.7604767680168152, -0.472320020198822, 1.7102227210998535, -0.41278910636901855, -0.06083350628614426, -1.3372613191604614, 0.1284891963005066, 0.8818211555480957, -0.27413007616996765, -1.1756625175476074, -0.6303789615631104, -0.0284982118755579, -1.8816261291503906, 1.0327072143554688, -1.0446429252624512, -1.686959981918335, 0.3563102185726166, 0.8824110627174377, 0.14727655053138733, 1.3784093856811523, 1.5799884796142578, 1.091314673423767, -1.583039402961731, -0.1063089668750763, 0.9109355211257935, 0.8288953304290771, 1.7721165418624878, -0.3320617973804474, -0.08088002353906631, -0.5810355544090271, -0.2180088460445404, -0.06052509695291519, 0.44231799244880676, -0.8125333786010742, -2.010373592376709, 0.32994234561920166, 1.4557217359542847, -1.2518129348754883, -0.013186217285692692, 0.3045894503593445, 0.11950083822011948, 1.569298505783081, 0.4523305594921112, 0.6955327391624451, -0.6745948791503906, -0.660936176776886, -0.6927865147590637, -1.3785152435302734, 0.9738855361938477, 0.13572710752487183, 0.9418838620185852, -0.3190089166164398, -0.9959372878074646, -1.295789122581482, 2.78997802734375, -0.9486463069915771, -1.4167134761810303, -0.11788804829120636, 1.0232139825820923, -2.4991211891174316, 1.1347935199737549, 0.9518038034439087, -0.2581075131893158, 1.1484850645065308, -1.2290107011795044, -0.1905147135257721, 1.6455159187316895, 2.526390790939331, 0.7409794926643372, 0.28307071328163147, 0.7293770909309387, 1.776404857635498, 1.045644760131836, 1.3887970447540283, -0.060874566435813904, -0.08271007239818573, -0.36678826808929443, 1.1125925779342651, 0.16432249546051025, 1.2563568353652954, 1.1083835363388062, 0.2203259915113449, 0.1857919543981552, 1.479049801826477, 0.7643568515777588, -0.5849335193634033, 0.32303640246391296, -1.3183737993240356, -0.9617506861686707, -0.28448379039764404, -0.23300744593143463, 0.6246893405914307, 2.3037827014923096, 1.2339091300964355, 0.16620919108390808, -0.9470763802528381, -0.36830779910087585, -0.41338953375816345, -0.43920764327049255, 0.10662342607975006, 0.3244282901287079, -1.1308268308639526, -1.2865383625030518, -2.2687582969665527, 0.21463897824287415, 0.5757613182067871, -0.3821306824684143, -1.828718662261963, 1.5959184169769287, -0.7393389940261841, -1.8726513385772705, -2.4063639640808105, -0.5737349987030029, -0.5091580152511597, 0.7273969054222107, -1.4510021209716797, -1.1448434591293335, 0.2836422026157379, -1.0546547174453735, -0.04529661685228348, 1.207783818244934, 0.7516273260116577, 0.5698409080505371, -0.3642899990081787, -0.8157400488853455, 2.0730111598968506, 0.9662895798683167, 0.10067318379878998, -0.6041083931922913, 1.043483853340149, -0.3943984806537628, -0.10562428086996078, -0.37506788969039917, 0.7711583971977234, 0.03505605831742287, 0.03729821741580963, 0.28531548380851746, 0.04382984712719917, 1.0298820734024048, 0.4934878647327423, -0.23581376671791077, 0.9909600615501404, 1.7649818658828735, -0.033627331256866455, -0.5366090536117554, 0.5455461740493774, 0.829662024974823, 0.15502667427062988, 1.2798349857330322, 0.17880061268806458, -1.0129640102386475, 0.7665188312530518, -0.13633207976818085, 1.4104403257369995, -0.6222991347312927, -0.33303797245025635, 0.4569052755832672, -0.24586904048919678, 1.4604347944259644, -1.6665018796920776, 0.40082478523254395, -0.11574430763721466, -0.3964347839355469, -0.5438617467880249, -0.7001386284828186, -1.5770901441574097, -0.4721076190471649, -1.6324137449264526, -0.813937246799469, -0.380790650844574, 1.13435959815979, 0.5894919037818909, 1.1949125528335571, -0.5080609321594238, 0.862160325050354, 0.5768901705741882, 0.5057162642478943, -0.8592658042907715, 0.8391014933586121, 0.04063224047422409, -0.2239207625389099, 0.21602821350097656, -0.5974836945533752, 0.0488007627427578, -2.323639392852783, 0.7812941074371338, -0.4226010739803314, 0.16889958083629608, 0.1437331736087799, -0.7660278081893921, 0.2527029812335968, 0.4535689353942871, 0.9014123678207397, -2.3315327167510986, 0.31779468059539795, 0.8265218138694763, 0.3733198046684265, -0.8758695125579834, 0.12887714803218842, -0.8950531482696533, -2.172699213027954, -0.06543075293302536, -1.5641037225723267, 0.24730046093463898, -2.9579851627349854, 0.5828346014022827, -0.5362000465393066, 0.6107867360115051, 0.681577742099762, 0.17916299402713776, 1.9461195468902588, -0.492319256067276, 2.8040707111358643, -0.9666206240653992, 1.554063081741333, -1.699135422706604, 2.467026472091675, 0.4696972966194153, 0.21688027679920197, 0.19747088849544525, -0.2391968071460724, -1.2565088272094727, 1.5086743831634521, 0.4009767472743988, -0.31946444511413574, -1.519670844078064, -0.3989097476005554, 1.5836329460144043, 0.31568241119384766, -1.0674810409545898, -0.7202355861663818, -1.1751607656478882, 1.1331546306610107, 0.6194993257522583, 1.2527493238449097, -1.2968498468399048, 1.2801276445388794, -0.054983898997306824, -2.3262670040130615, 0.3141885995864868, 2.3551251888275146, -0.11898291856050491, -0.3036603629589081, -0.9838532209396362, -0.3316502869129181, 0.6720881462097168, -0.767992377281189, -0.0052634612657129765, 0.05174781009554863, -0.16876830160617828, 0.8538529276847839, 0.8062400221824646, 0.07329647988080978, -0.14867380261421204, 0.8683633208274841, -0.5450869202613831, -0.6845923066139221, -2.4584991931915283, 1.170720100402832, 1.8031365871429443, 0.03524458780884743, -0.6873987913131714, -0.11644431203603745, 0.09441262483596802, 0.8943155407905579, 1.136289119720459, -1.0802838802337646, -0.1669129580259323, 0.46682047843933105, 0.5242931842803955, 1.5542211532592773, -0.3382958173751831, 1.9414875507354736, 0.25548484921455383, -0.744166910648346, -0.5623660683631897, 0.2321438491344452, 2.253807783126831, -1.656154990196228, -0.5361186265945435, -1.7705636024475098, 0.4905086159706116, -0.07766669988632202, 1.5194705724716187, 1.45702064037323, -0.26982763409614563, -0.36737120151519775, -0.7770640850067139, 1.6187578439712524, 0.539843738079071, 0.3788087069988251, -0.7035350203514099, -0.07116708159446716, 1.3685885667800903, 0.030846308916807175, -0.78996342420578, -0.43544718623161316, 0.9654257893562317, -0.3514518737792969, -0.6257038116455078, 0.07296337932348251, 0.18347392976284027, -1.746363878250122, -1.1073169708251953, 0.800602376461029, 0.6641515493392944, -1.3742148876190186, 0.3964708745479584, -1.2509232759475708, 1.413158655166626, -1.0975186824798584, -1.2289707660675049, 0.552068293094635, -0.44604888558387756, 2.340092897415161, 0.05582983419299126, 1.197072982788086, -0.2728012502193451, 1.2642590999603271, -0.8009305000305176, 0.5403043627738953, 0.6569872498512268, -1.0527137517929077, 1.4893139600753784, -0.9098077416419983, 0.027910083532333374, -0.9614708423614502, -1.2308270931243896, -0.2576032280921936, 0.7732706665992737, 0.1153232604265213, 0.9721613526344299, -0.46170544624328613, -0.5301942229270935, 0.03278322517871857, 0.6066279411315918, -0.645334780216217, 2.064819574356079, -1.126423954963684, 1.5227289199829102, -0.5660292506217957, 0.06748689711093903, 0.8975372910499573, 0.025996578857302666, -0.15911424160003662, -0.45906639099121094, 1.2588163614273071, -2.2097039222717285, 0.9083824753761292, -0.978872537612915, 1.127191424369812, -0.8536510467529297, 1.01261568069458, 0.3675178289413452, 0.10634993761777878, 2.1391868591308594, -0.592734694480896, 1.6223875284194946, 0.019992131739854813, 0.7089466452598572, 0.3426622748374939, -0.07905963063240051, -2.808067798614502, 0.24305884540081024, 0.32256263494491577, -0.5860868096351624, -1.358487606048584, -1.3082878589630127, 0.9997290968894958, 0.9781519770622253, -0.3233678638935089, 1.3274271488189697, -0.2985273599624634, -0.9038345217704773, -0.52109694480896, -0.279598593711853, 0.2981403172016144, 1.0763037204742432, -0.9012172818183899, 1.8657997846603394, -1.324616551399231, -1.345104694366455, 0.8847923874855042, 0.1957685947418213, 1.337180733680725, 0.38848283886909485, 1.8252500295639038, 0.7694784998893738, -1.8068386316299438, 0.3406088352203369, -0.14530859887599945, 0.2729716897010803, 0.7982681393623352, 0.1677040308713913, 0.22491870820522308, 0.3791939318180084, 0.28289321064949036, -0.165657639503479, 0.6793205142021179, -0.0008086949237622321, -0.04506073147058487, 0.31042933464050293, -0.43369466066360474, 0.5486137866973877, 1.1322909593582153, -0.5282222032546997, -0.52252197265625, -0.9111389517784119, -1.479047179222107, -0.6212582588195801, 0.5732083320617676, 1.3363090753555298, -0.20450004935264587, 1.0535880327224731, 1.4314219951629639, -0.4836886525154114, 1.1660311222076416, 1.175645351409912, -0.8618245720863342, 1.9849637746810913, -0.26663118600845337, -0.7952647805213928, 0.39239752292633057, 0.17753131687641144, 0.9175973534584045, -0.7496654987335205, 0.09187708050012589, 0.069730244576931, -0.24998041987419128, 1.1970075368881226, 1.1917643547058105, 0.3675290048122406, 0.03182706981897354, -0.4021257162094116, 1.3083875179290771, -0.7593428492546082, -0.3208886682987213, -0.6835439205169678, 1.0131065845489502, 0.3766389787197113, 0.06191127374768257, 0.7383254766464233, 0.7510684132575989, 0.4288472533226013, -0.4894750714302063, 1.371343970298767, -0.17592814564704895, -1.9117993116378784, 0.9518190026283264, -1.2578483819961548, -1.2559908628463745, 1.0356026887893677, 0.16926859319210052, 0.42226889729499817, -0.40011855959892273, -0.2920112609863281, 0.3165417015552521, -1.1975950002670288, 0.699226975440979, 0.0882965698838234, -0.052409179508686066, 0.36133477091789246, -0.038475509732961655, 0.6271776556968689, 0.3512386381626129, -1.0004931688308716, -0.9853667616844177, -1.218263864517212, 1.509892225265503, 1.1754341125488281, 0.6010985374450684, 0.8469715714454651, 0.026067111641168594, 0.06234583631157875, 0.06663759052753448, -0.15526174008846283, 0.4137287735939026, -0.6719217300415039, 1.4494651556015015, 0.4057605266571045, 1.6204681396484375, -0.018848348408937454, -2.2353711128234863, 1.5627714395523071, -0.5563486218452454, 0.2807866930961609, -1.5885590314865112, 2.551952838897705, 1.4152461290359497, 2.607841968536377, 2.146479368209839, 1.4193696975708008, 0.4742231070995331, -1.819532036781311, -1.9494528770446777, -0.8654971718788147, 0.7541664838790894, 1.1712878942489624, 1.0573559999465942, 0.3868563771247864, 0.9541714787483215, 0.15196266770362854, 2.648982286453247, -0.4762451946735382, 0.5713325142860413, 0.6690197587013245, 0.3158003091812134, 1.038532018661499, -1.6902135610580444, -0.4084644913673401, -1.1216896772384644, -0.8361480236053467, 0.38289541006088257, 1.3284858465194702, 1.1499987840652466, 0.29187875986099243, -1.1958961486816406, 0.2879379093647003, 0.38313066959381104, -0.1319497972726822, -0.1258477419614792, 1.3269319534301758, -1.0252286195755005, 0.7121545672416687, 0.30050936341285706, 0.22624574601650238, -0.6536497473716736, -0.35750770568847656, -1.4127832651138306, 1.1176114082336426, 0.07650915533304214, 0.4733705520629883, -0.06331437081098557, 1.2742160558700562, -0.10936406254768372, -3.391413450241089, -0.25759434700012207, 1.3536498546600342, 0.08470389991998672, 1.1632972955703735, -0.9635977745056152, 0.8160532116889954, -0.5936017632484436, -1.2993693351745605, 0.2153460681438446, 0.0010689834598451853, -0.988018810749054, -0.669884979724884, -0.3410414159297943, -0.555499792098999, -0.25213760137557983, 0.5657253861427307, 0.8377913236618042, -1.0337966680526733, 0.684309720993042, -0.17229028046131134, 0.6707252860069275, 1.0754421949386597, 0.30650922656059265, -0.8414115309715271, 1.4821873903274536, 0.2549263536930084, -0.10171347111463547, 1.1038013696670532, -0.1977960169315338, -0.12130933254957199, 0.6889570355415344, 0.9484891891479492, 1.4438775777816772, -1.876318097114563, -0.2703145146369934, 1.4430197477340698, 0.27195674180984497, -0.05749206244945526, -1.2318757772445679, 0.7323459982872009, -0.12851153314113617, -0.32956966757774353, 0.21458229422569275, 0.011427387595176697, -0.4932209551334381, 0.9831504821777344, 0.19111137092113495, 0.1737644225358963, -0.30287671089172363, 0.28022363781929016, 1.9836807250976562, -0.8092536926269531, -0.022195497527718544, 2.2513723373413086, -0.9359358549118042, -0.3337382674217224, -0.598095178604126, 0.6024354100227356, -0.2773284912109375, -1.8899520635604858, -0.08216604590415955, -0.029348645359277725, -1.9540103673934937, -0.6239851713180542, -1.817116141319275, -0.893903374671936, -0.6373786926269531, 1.6719671487808228, -0.12035417556762695, -1.053553581237793, -0.7134082317352295, 0.44461485743522644, 1.0311880111694336, -0.014280674047768116, 0.05948713794350624, 0.5082426071166992, -0.3964696526527405, -0.03182728588581085, -1.2765733003616333, -0.5128871202468872, -0.9357249140739441, 1.2482088804244995, 0.39395004510879517, 1.415439248085022, 1.0699818134307861, 0.32208284735679626, -2.166750192642212, 1.191498041152954, 2.483323574066162, -0.14160576462745667, 1.5739909410476685, 1.1576073169708252, 1.1723066568374634, -1.609985113143921, 2.0161638259887695, 1.157447099685669, 0.630375862121582, 0.39086538553237915, 3.1927402019500732, -0.5812934041023254, 1.349752426147461, 0.0026829889975488186, -1.0109999179840088, 1.3134499788284302, 0.7174093127250671, 3.3254573345184326, 1.3913774490356445, -1.102957010269165, 0.3155041038990021, -0.04812176525592804, 1.601879596710205, 1.3438246250152588, -0.9418633580207825, -0.1310996562242508, 0.9286766052246094, -0.26848816871643066, -1.6850076913833618, 0.5145537853240967, -0.3117940127849579, -0.6221182942390442, -0.24400827288627625, -0.0901261568069458, -0.942894458770752, -0.8056818842887878, -0.8558711409568787, 0.87247633934021, 0.6521696448326111, -1.7765330076217651, -0.3767804801464081, 2.131225347518921, -0.07206222414970398, -0.9148926734924316, 2.066781997680664, 0.551571249961853, 0.45691362023353577, 0.21945860981941223, 0.5581702589988708, -0.8910560011863708, -0.9588321447372437, -0.1633712202310562, 0.3758206069469452, -1.227475643157959, -0.3943638205528259, 0.7789929509162903, 0.7161015868186951, 0.6213851571083069, -0.9493169784545898, -0.01774708554148674, 1.40794837474823, 1.7239688634872437, -0.6657387018203735, 0.5485787987709045, -0.551937460899353, 0.13510455191135406, -1.2571686506271362, 0.3697699308395386, -0.28399327397346497, -0.7367295026779175, -0.35784676671028137, -0.2642868161201477, -0.3357912600040436, 1.5343446731567383, 2.1460020542144775, 0.6283589005470276, -0.03053675964474678, 0.5476117134094238, -0.06527871638536453, 1.2052652835845947, 1.9879788160324097, -2.2365522384643555, -0.3856629729270935, 0.21004115045070648, -0.5572982430458069, -0.5428950786590576, 1.1682265996932983, -1.316503882408142, 0.5298817157745361, 1.649560570716858, -0.08768755942583084, -1.130967617034912, -0.4056423604488373, -1.0634459257125854, 0.4963698983192444, -1.2568819522857666, -0.5354689359664917, 1.3565831184387207, -0.48235276341438293, -0.7553310394287109, 1.2668730020523071, -0.4243859350681305, -1.9425439834594727, 0.4855886995792389, 0.27650681138038635, -0.9270082116127014, -1.2054815292358398, -0.18888933956623077, 0.5741717219352722, -1.5175504684448242, -0.6088846325874329, -0.021183060482144356, -0.13398543000221252, -0.7225058674812317, -0.3143637776374817, -0.3890942931175232, 1.0018737316131592, -1.5433958768844604, -0.9131922721862793, -0.7563002705574036, -0.48646533489227295, -0.5977649688720703, -0.9153134226799011, -1.5118670463562012, 0.44736114144325256, -0.2724616229534149, 2.628309726715088, 0.6508125066757202, -0.5332870483398438, -0.23977547883987427, 1.638375163078308, 1.080083966255188, 0.661141037940979, -1.1242204904556274, -1.7739204168319702, 0.1872708946466446, 0.15825963020324707, 1.3124550580978394, -0.2387063056230545, 0.9677916765213013, 1.293826937675476, -0.06110500916838646, 0.9056085348129272, 0.5257302522659302, 1.262699007987976, 0.39442768692970276, 0.9063008427619934, -0.9589407444000244, -1.147925853729248, 3.2184925079345703, -0.6781047582626343, 0.9259183406829834, 0.671582818031311, -0.26991328597068787, 0.5085119605064392, -1.061203122138977, -0.13068024814128876, -0.34250855445861816, -0.34828656911849976, -0.6729540228843689, 0.9476821422576904, -0.6623585820198059, -1.531180739402771, 1.073649525642395, -0.793123185634613, 1.3723855018615723, -0.46931177377700806, -0.25685927271842957, 0.45704707503318787, -0.18514922261238098, 0.5328397154808044, 0.2722175419330597, 0.596049964427948, -0.07897572219371796, 0.8824439644813538, 0.15817104279994965, -0.5470131635665894, 0.04379059374332428, -1.0484323501586914, 0.6492214798927307, -1.230393409729004, -0.9249172210693359, -0.17896750569343567, 0.09416987746953964, -1.131381630897522, -0.7343423366546631, -0.4099670648574829, -0.17892992496490479, 1.1425976753234863, 0.4081166386604309, 1.3528900146484375, -1.0294606685638428, 0.9607509970664978, 1.7693418264389038, -0.4821373224258423, 0.7668755054473877, -0.011815352365374565, 0.08907979726791382, -1.2395321130752563, 1.7045483589172363, 3.9065754413604736, -1.1826125383377075, 1.942348837852478, -0.03002188913524151, 0.17704123258590698, 1.0177544355392456, 0.45740920305252075, -2.4734339714050293, -0.17718087136745453, -0.10813194513320923, 0.5190110802650452, 0.11089633405208588, -0.6208486557006836, 0.9243350625038147, 1.9853051900863647, 1.233386754989624, -0.8742731809616089, -0.6309731602668762, 1.0690340995788574, 1.0449048280715942, -0.09740573912858963, -1.6521327495574951, -1.2230794429779053, 0.21724797785282135, 1.2472602128982544, 2.0816171169281006, 0.8973526954650879, 0.14144451916217804, -0.4433651268482208, 0.2706664800643921, -0.9947795271873474, 1.2253907918930054, 0.25630342960357666, -0.6121773719787598, -0.04195169359445572, -0.45963743329048157, -0.055883582681417465, -0.9300518035888672, -0.604354739189148, 0.25367236137390137, 0.9481699466705322, -1.4876292943954468, -1.1993954181671143, 1.5746021270751953, -2.3315892219543457, -0.8390324115753174, -0.34926968812942505, 1.019949197769165, -0.4738216996192932, -1.3744804859161377, 0.18863534927368164, 0.04918856918811798, 1.3796334266662598, 0.2072126269340515, -1.9084548950195312, 0.13086122274398804, -0.7317343950271606, 0.2207670956850052, 1.9959830045700073, -0.16182543337345123, 1.1193645000457764, 2.5712459087371826, 1.5416419506072998, -0.5456997752189636, 0.884666919708252, 1.049162745475769, -0.8175708055496216, 2.143378734588623, -0.741100549697876, 1.5051031112670898, 0.15699340403079987, -0.43746790289878845, 0.6348415613174438, 0.13645413517951965, -0.12805752456188202, 1.2574149370193481, -1.5235400199890137, 0.0864538624882698, 0.16085787117481232, -0.9444403648376465, 0.24291974306106567, -1.3055981397628784, -0.40796133875846863, 0.5880256295204163, 0.8253968954086304, 1.6957510709762573, 0.4777573049068451, -0.4864788353443146, 2.1529974937438965, -1.6062999963760376, -0.15821295976638794, 0.7185133695602417, 1.514072060585022, -0.6569004058837891, 0.9010493159294128, 1.202795386314392, 0.20830672979354858, 0.6556782722473145, -0.9068578481674194, 0.9551786780357361, 1.477643370628357, 0.061827052384614944, -0.48854443430900574, 0.9269132614135742, -0.6045361161231995, 1.4417433738708496, 1.7318058013916016, 1.3091918230056763, -0.24354423582553864, 0.08812735229730606, 1.2969295978546143, 0.7483561038970947, -0.6021938323974609, -0.08194874227046967, 0.8198509216308594, -0.568316638469696, -1.1152831315994263, 1.9773505926132202, 0.0623023547232151, -0.2321542650461197, 0.5930243134498596, 0.7772778868675232, -0.19485390186309814, -0.15914377570152283, 0.46888744831085205, -0.4254147708415985, -0.7909963130950928, -1.7575368881225586, 2.035315752029419, -1.0432860851287842, 1.0052169561386108, 0.8470394611358643, 0.8779729604721069, -0.8555954098701477, 0.03000662662088871, 0.2920163571834564, 0.2997063994407654, -0.6921019554138184, 0.29088497161865234, 0.6404786705970764, -1.1216812133789062, 0.6081863045692444, -2.4311130046844482, -0.1275518834590912, -2.207341432571411, 1.2645010948181152, -0.488877534866333, 0.7496397495269775, -0.8185490369796753, -0.18734316527843475, -0.9976105690002441, 1.3820124864578247, 1.8094909191131592, -0.05060417577624321, 0.5880098938941956, 0.9652719497680664, 0.9046555161476135, 0.8484171628952026, 0.32836902141571045, -1.5186944007873535, -0.5530908107757568, 0.14564669132232666, 0.716874361038208, -0.8963783979415894, -1.858721375465393, 2.72309947013855, -0.9530938863754272, -0.21159657835960388, 0.48056358098983765, 0.8059362769126892, -1.2299082279205322, -0.6693846583366394, -0.022534113377332687, -0.2709383964538574, -0.33614298701286316, -0.10924313962459564, -0.4740164279937744, -0.9465204477310181, -1.4539324045181274, 0.7733983397483826, -0.9500275254249573, -0.1991579234600067, -0.37124335765838623, -0.6938588619232178, 0.02657511830329895, -1.207729458808899, 0.05748631805181503, -0.997146487236023, -1.0219398736953735, 0.03544261306524277, -0.2652716338634491, 0.3643403649330139, -1.304310917854309, 0.23415684700012207, 0.5981485247612, -0.35261908173561096, -0.21341143548488617, -0.6544402837753296, 0.17833945155143738, 0.8336470723152161, 1.3431707620620728, -1.5401878356933594, 0.47040989995002747, 0.39270827174186707, 1.9995583295822144, -0.0807952806353569, 0.4480860233306885, -0.2657034397125244, 2.1409592628479004, -1.235547661781311, 0.9642096757888794, -0.018671827390789986, 0.3656664490699768, -0.15586699545383453, 0.4805585443973541, -0.9210078120231628, -2.6278507709503174, 2.5496931076049805, 0.005935055203735828, -0.46373701095581055, -1.2975746393203735, -1.0367553234100342, -2.21559739112854, 1.3496366739273071, -0.44704365730285645, 0.12789587676525116, -0.44821277260780334, -1.0628329515457153, -0.42091843485832214, -0.8685920238494873, 0.3844996988773346, 0.05570201948285103, -0.042759090662002563, -0.7670208811759949, 0.4943861663341522, 0.6799913048744202, 0.9982398152351379, -0.9947280287742615, 0.9383995532989502, -1.6566253900527954, 0.5929868817329407, -1.2797598838806152, -1.0210336446762085, -2.213294744491577, 1.0516008138656616, -0.9030901193618774, -2.2484397888183594, 1.0430874824523926, 0.5283793807029724, 0.8385611176490784, -1.023903727531433, -0.5903439521789551, 0.6012613773345947, -0.5898663401603699, 0.9568302035331726, -0.43945276737213135, -0.6045883297920227, -0.5006972551345825, 0.8979273438453674, -0.8094447255134583, 1.4345810413360596, 0.651024580001831, 1.7633146047592163, -0.9504519701004028, -0.271171897649765, -0.9973727464675903, 0.2586846947669983, -0.5017490983009338, 0.35931694507598877, 0.7445248961448669, -0.08846363425254822, 1.9287692308425903, -0.1230306401848793, 1.0293011665344238, -0.7758191823959351, 0.12878617644309998, 0.9619888067245483, -1.8601791858673096, 1.7823153734207153, 0.6560080051422119, 0.4295738935470581, 0.7162919640541077, -1.8282090425491333, -0.018564896658062935, -0.11892376095056534, -0.9817420840263367, -0.33720147609710693, -0.5584213733673096, -0.0015810664044693112, -2.4508378505706787, -1.0293643474578857, 0.013540658168494701, 1.4047222137451172, 0.15156672894954681, 1.0332441329956055, 0.12009529769420624, -0.43112871050834656, -0.6737098097801208, -0.0282637607306242, 1.8195006847381592, 1.3994884490966797, 0.4392361342906952, -2.104304552078247, 1.686773657798767, 0.6858843564987183, 0.40704506635665894, 0.677914559841156, -0.9132474064826965, 0.004760894924402237, 0.3288895785808563, -1.1862767934799194, -0.34655413031578064, -0.20636717975139618, 1.7676461935043335, -0.4712144732475281, -0.6669069528579712, 0.9768766760826111, 0.25985148549079895, 0.9367820620536804, -0.4866001009941101, 0.07031948119401932, 0.08565045893192291, -0.7496758103370667, -0.29451796412467957, -0.2809397280216217, -2.7188186645507812, -3.421048879623413, 0.4439442455768585, 0.674993634223938, -0.5361945033073425, -0.16048744320869446, 0.43514639139175415, 1.852588415145874, -0.18544436991214752, 0.3894141614437103, -1.0920865535736084, 1.1370439529418945, 0.5557866096496582, -0.9062309265136719, 0.9156282544136047, -1.192042589187622, -0.5352351665496826, 0.8119566440582275, 1.3822431564331055, -0.06737996637821198, -0.4403434097766876, 0.10915114730596542, -0.2172154188156128, -0.027140866965055466, 0.5039420127868652, 0.9700084924697876, 0.4816296696662903, 1.1572386026382446, -0.9102766513824463, 1.525375485420227, 0.9847257733345032, -0.25427210330963135, -0.36865806579589844, -0.7463054060935974, -0.09784317761659622, -1.6690208911895752, 0.22746005654335022, -1.4089058637619019, -0.32843634486198425, -1.261568307876587, -0.39688587188720703, 0.028108661994338036, -0.6228126287460327, -0.2898768484592438, 1.5997800827026367, -0.2628301680088043, 0.9779992699623108, 0.5280701518058777, 0.2933874726295471, 0.45137473940849304, -0.26215648651123047, -0.7096350789070129, 0.29595473408699036, -0.16650596261024475, -0.2913067042827606, 0.7257257103919983, -0.07702738046646118, -0.12384269386529922, 2.0671379566192627, 1.7681281566619873, 1.498212218284607, -0.6419981122016907, 1.1325465440750122, -1.4571614265441895, -0.9099966287612915, 0.5277962684631348, -2.188598394393921, -0.1575174480676651, -0.34353312849998474, -1.138153076171875, 1.0580997467041016, -0.10086781531572342, 0.4884122908115387, -1.38637113571167, -1.2671700716018677, 1.4335963726043701, 0.3853890895843506, -0.9100576639175415, 1.3517895936965942, 0.729476273059845, -1.0752042531967163, 0.4649510681629181, -0.12694254517555237, -0.8419642448425293, -0.912154495716095, -0.7675715684890747, 0.05462171137332916, -1.382104754447937, -0.8690273761749268, 0.6443690657615662, 0.8422063589096069, -1.8963111639022827, 0.44742730259895325, -0.6244248151779175, -0.9037641882896423, 1.1501566171646118, 0.3661643862724304, 0.1382589340209961, -0.002600703854113817, -0.3907822072505951, 0.13164126873016357, 0.5640061497688293, -1.1916764974594116, 0.682824432849884, 1.124679684638977, -0.03569502755999565, 0.22974437475204468, 1.1322022676467896, 0.2907909154891968, 1.0476741790771484, -0.6562737822532654, -1.840775489807129, 0.43835294246673584, 0.9778735041618347, -0.2936423420906067, 2.352020263671875, -0.44736406207084656, -2.4943902492523193, 0.38711321353912354, 0.7234074473381042, -1.9536792039871216, -0.6106313467025757, -0.17022567987442017, 0.08744502067565918, -1.776410460472107, 1.718460202217102, 0.6810427308082581, -1.0212262868881226, -0.6921772956848145, -0.6307689547538757, -0.0423738993704319, 0.5559182167053223, -0.5834449529647827, 0.6743927597999573, -1.081047773361206, -0.802614688873291, 0.06866458803415298, 0.07349956780672073, 0.04797784611582756, -1.2359145879745483, 0.6326936483383179, 0.8560073375701904, 2.0166423320770264, 0.061766140162944794, -0.4230457544326782, -0.28388547897338867, -1.5765869617462158, 2.2428946495056152, -0.5253207087516785, -1.341378927230835, -1.9341766834259033, -2.6716718673706055, 1.9502017498016357, -1.3584996461868286, -0.2434462308883667, -1.3422154188156128, 1.4300111532211304, -1.4742289781570435, 0.844006359577179, -1.0368187427520752, 1.0053939819335938, 0.37184661626815796, 0.12528011202812195, 1.1237584352493286, -0.3994734287261963, 1.2398649454116821, -0.4193485379219055, 1.8191571235656738, 1.558246374130249, -0.13446356356143951, -1.0733145475387573, 0.5053298473358154, 0.6878432035446167, 0.013919062912464142, 0.12384574115276337, 1.488326072692871, 0.1486697942018509, -0.5349751114845276, 1.0694926977157593, 0.6960859298706055, 0.7055765986442566, -0.2861555218696594, -0.23481601476669312, -0.3100275695323944, 0.6480212807655334, 0.5929811000823975, 0.008750427514314651, 0.4436439871788025, -1.6057554483413696, 1.4489085674285889, -0.18612627685070038, -1.4989100694656372, -0.5269847512245178, 0.1971871256828308, -0.18623748421669006, -0.3961544930934906, -0.7581121921539307, 0.9134637713432312, -0.7630305886268616, -0.3757021427154541, 0.5530399680137634, 0.43473145365715027, 1.738358497619629, 0.2052128165960312, 0.015549831092357635, -0.13742467761039734, -0.3726601004600525, 1.0012820959091187, -1.1745201349258423, 1.4581841230392456, -0.1157948449254036, -0.4373285472393036, 1.174480676651001, -1.6681195497512817, -2.259716033935547, 0.441644549369812, -0.5453998446464539, -0.718796968460083, -0.7071208953857422, 2.5965466499328613, -0.3408818244934082, 0.22732491791248322, -0.47155770659446716, 0.9817019104957581, -2.254185914993286, 0.01026537548750639, 0.15870709717273712, 0.16378958523273468, -0.6817684769630432, -0.5911071300506592, -1.3413610458374023, 0.35489514470100403, 0.7071198225021362, 0.563640296459198, 0.4194454252719879, 0.8077433109283447, -0.5644192695617676, -2.0261242389678955, -1.1559553146362305, 0.6748518943786621, -1.614227056503296, -0.33037954568862915, 0.7956116795539856, 0.4691131114959717, 0.3756594955921173, 0.12572580575942993, -0.19879822432994843, -0.5436007976531982, 1.0954660177230835, -0.6475323438644409, -0.38374385237693787, 0.3564468026161194, -0.9870404005050659, 1.0959246158599854, -0.7061366438865662, 0.5142384767532349, 0.4649865925312042, 0.5511168837547302, -1.962419867515564, -0.1752261370420456, -1.3275213241577148, 1.1064823865890503, -0.3991255462169647, -0.7628606557846069, -0.002629605820402503, -1.3137538433074951, -0.5771215558052063, 0.802432656288147, -0.08431243151426315, -0.3029020428657532, -0.34957951307296753, -1.3556073904037476, 0.1275235116481781, 0.2169194370508194, 0.48584792017936707, 1.5716564655303955, 0.4374292194843292, -0.3085983991622925, -0.9349175691604614, 1.221696376800537, -1.460432529449463, -1.8829360008239746, 0.04460615664720535, 0.6802930235862732, -0.17795397341251373, 0.4738124907016754, 0.8559793829917908, 0.2493342012166977, 0.557230532169342, 0.339269757270813, 0.22931674122810364, 0.5362313985824585, 0.41445934772491455, 1.0067310333251953, 0.15056589245796204, 0.2760660648345947, -0.1620364636182785, 0.3553749620914459, 0.2402111440896988, 0.07863610237836838, -0.3969539403915405, -0.4771097004413605, 0.28351113200187683, -1.1647883653640747, 1.498373031616211, -1.0288753509521484, -0.30239760875701904, -0.2524484694004059, -1.2141673564910889, 1.9489558935165405, 0.3933255970478058, 0.20358823239803314, -0.22180145978927612, -0.28870636224746704, 0.2958439588546753, 0.1281808316707611, 1.7494152784347534, -0.3089979588985443, 1.214882731437683, -0.9814590215682983, -0.3422603905200958, -2.1367740631103516, -0.6467220783233643, 0.5106819868087769, -0.05194223299622536, -0.8428699374198914, -0.06244993209838867, 0.0033359676599502563, 0.737392008304596, 0.4398391842842102, 1.1883697509765625, -0.49714991450309753, -2.155665636062622, 0.48407790064811707, -0.542877197265625, 0.6977649927139282, 0.22334259748458862, 2.156930446624756, -0.9470216035842896, 0.5213946104049683, 0.878435492515564, -0.7412397861480713, -0.43344688415527344, 0.6397995948791504, -0.9268394708633423, -0.5323318243026733, -0.28972509503364563, -0.48805326223373413, 2.128096580505371, -0.44771671295166016, -0.17407451570034027, 1.3045259714126587, -1.3710154294967651, 1.0378222465515137, 1.1037472486495972, -0.5205230116844177, -0.882476270198822, 1.4785679578781128, -1.1458251476287842, 1.3135823011398315, -0.7311722040176392, 0.6484338641166687, 0.49036848545074463, -0.4711792469024658, -0.7880083918571472, 0.7780317664146423, 0.4536241888999939, 0.1668097972869873, 0.4313453435897827, 0.8795941472053528, -0.39947402477264404, 0.6127195358276367, -0.29536065459251404, -0.7783926129341125, 0.845238208770752, 0.19942353665828705, 1.6123219728469849, -2.8463690280914307, -2.00535249710083, -0.686555802822113, -0.4530315101146698, -0.04518552124500275, 0.6790320873260498, -0.1354692429304123, -1.3950016498565674, -1.5138219594955444, -0.0865895226597786, -0.044502727687358856, -1.0055469274520874, -0.8830639719963074, 1.3036445379257202, 1.2979971170425415, -0.780258297920227, -0.06529895216226578, -1.2160258293151855, 0.9804141521453857, 2.6558706760406494, -0.7529322504997253, 0.9693494439125061, -0.13929729163646698, -1.7261326313018799, -2.3385345935821533, 0.8997415900230408, 1.8707311153411865, 0.19717448949813843, -0.62501460313797, 0.872579038143158, -0.4710133969783783, -1.8484591245651245, 1.327553391456604, 1.4425112009048462, -0.39662736654281616, 0.9135845303535461, -1.344123363494873, -0.8965219855308533, 0.20494972169399261, -1.1546127796173096, -2.08966326713562, -0.854852557182312, -0.4109383821487427, 0.16618837416172028, -0.5670083165168762, -0.8748676180839539, -0.6056921482086182, -0.6751745939254761, 0.10732252895832062, -1.8899662494659424, 0.9094662070274353, 0.4487168490886688, -1.6866719722747803, -0.20587025582790375, 0.40071508288383484, 0.45063456892967224, -0.2581254243850708, 1.4587277173995972, -0.5468409061431885, -0.4933532774448395, -0.6569746136665344, -1.5347365140914917, 0.4998664855957031, 0.6615411639213562, -0.9343660473823547, -0.0939956083893776, 0.00820013228803873, -0.21018528938293457, -2.0609633922576904, 1.2049212455749512, -0.8054609894752502, 0.5878974199295044, 0.11364945769309998, -1.6185606718063354, -0.8246118426322937, 2.094622850418091, -0.5373024940490723, 1.5218284130096436, -1.3227087259292603, 0.36004894971847534, -0.2068040370941162, 0.10996207594871521, -0.9977946281433105, 0.09560966491699219, -0.8937236070632935, -0.42655694484710693, 0.5715469717979431, -0.5706710815429688, 1.2749145030975342, 0.1134335920214653, 0.577128529548645, -0.8084589838981628, 0.02612069807946682, 0.6580536961555481, -0.025456208735704422, 0.27413272857666016, 0.2049674391746521, 0.36215436458587646, -0.3947460949420929, -1.4581977128982544, -0.6004248857498169, 0.024496452882885933, 0.6105342507362366, -0.35016095638275146, 0.7116587162017822, -0.21071524918079376, 0.17751650512218475, 0.03575698286294937, 0.11334330588579178, 0.3886101543903351, -0.8187267780303955, -0.19921445846557617, -0.012273979373276234, 0.8466311097145081, -0.7642927765846252, 0.6620379090309143, -1.0394105911254883, 0.07075059413909912, -1.35062575340271, 0.8056504130363464, -0.6796668171882629, -0.0005717206513509154, -1.3161624670028687, 0.5648719072341919, -1.0511475801467896, -1.2190418243408203, -0.618215799331665, -0.1221105083823204, -0.9597739577293396, -2.8020389080047607, -0.9068094491958618, 0.5816366076469421, 1.6803170442581177, 0.04028225317597389, -0.8344892859458923, 0.7099761962890625, -0.06422963738441467, -1.107647180557251, -0.3771013617515564, 1.067640781402588, -0.281171590089798, 0.03895235434174538, -0.41382095217704773, 0.05960100144147873, 0.49773019552230835, 0.3180491030216217, 0.21219556033611298, -0.7839180827140808, -1.1944040060043335, 0.25244706869125366, 0.9717355966567993, -0.21340976655483246, 0.5954524278640747, -1.309627652168274, -0.02769818715751171, 0.8735805153846741, -0.12030457705259323, -1.671346664428711, 2.1562325954437256, 0.7813946008682251, 0.5538889169692993, -0.2653355896472931, 0.45745956897735596, 0.9896189570426941, 0.35827818512916565, 1.3975249528884888, 1.227983832359314, 0.04292307049036026, -1.2796238660812378, -0.3462028205394745, 0.017199033871293068, -0.788148045539856, -0.08110706508159637, -0.052298858761787415, 1.118095874786377, 0.0035420237109065056, -0.7818963527679443, 0.5177190899848938, 0.6433570384979248, -0.3158281445503235, 0.13782502710819244, 0.1671142578125, 1.2212101221084595, 0.6047263145446777, 0.23427186906337738, -2.514798879623413, 0.4303375780582428, -0.47909218072891235, -1.683529019355774, 1.2690597772598267, -1.1530323028564453, -0.7655835151672363, -0.21294094622135162, 0.5213974714279175, 0.6257192492485046, -0.29373008012771606, -0.7469061613082886, -1.2918297052383423, -0.965975284576416, 1.8841270208358765, -0.6269090175628662, -1.3147622346878052, -0.40020617842674255, -1.8527590036392212, 0.21676960587501526, -1.178279161453247, -0.352382093667984, 1.5057611465454102, 0.4871343672275543, -0.37886786460876465, 0.5627255439758301, -0.6688172221183777, -0.31842875480651855, -1.364943504333496, 2.9611010551452637, -1.1030066013336182, -0.6773484349250793, 0.3399587869644165, -0.8045053482055664, 0.0830591544508934, -1.738390564918518, -1.0885076522827148, -0.2409709393978119, 0.13826227188110352, 0.37794050574302673, -0.4313625991344452, -0.07613261789083481, -0.4427393078804016, -0.9431394338607788, -0.8771867156028748, 0.4022543728351593, -0.47843822836875916, -0.6813532710075378, -0.08815360814332962, -1.5890522003173828, 0.40283164381980896, 0.46884849667549133, 0.8705736398696899, 0.5946560502052307, 0.01567268744111061, 1.7084739208221436, -0.7193080186843872, -0.1357499361038208, 1.6346863508224487, -0.6983682513237, -0.2488141506910324, 2.007208824157715, -0.40525227785110474, -1.6470102071762085, 2.2662086486816406, 0.09554456174373627, 0.47439029812812805, -0.021062450483441353, -1.4751238822937012, 0.5820277333259583, 0.35903462767601013, -1.0734401941299438, -0.9882268905639648, -0.34077373147010803, 1.5061851739883423, -0.4517654478549957, -0.8378095626831055, 1.0933536291122437, -0.7132306694984436, 0.40817686915397644, 0.22569747269153595, -1.4839973449707031, -0.05757225677371025, 1.1456005573272705, 1.5254852771759033, 2.203890562057495, 1.1367021799087524, -0.9664758443832397, -0.6074729561805725, 0.3637300729751587, -0.6913390755653381, -0.05038497969508171, -0.8473889827728271, -0.6745581030845642, 0.29162290692329407, -0.04030865058302879, -0.6629599332809448, 0.2600463926792145, 0.5858387351036072, -0.5093259215354919, -1.978222131729126, 1.4388078451156616, -0.11661873757839203, 0.3356827199459076, 0.027143599465489388, -0.5834814310073853, -0.8019217252731323, -0.8112388253211975, 0.21306821703910828, 0.3834458887577057, -0.924968421459198, 0.10887004435062408, 1.5579495429992676, 0.48931586742401123, -0.17428579926490784, 0.6493939161300659, -0.6548128724098206, 1.4794052839279175, -0.2548462450504303, -1.7622309923171997, 0.6773403286933899, -1.4627140760421753, 0.44892388582229614, 0.7695149779319763, -1.387662410736084, 0.32345178723335266, -1.5236448049545288, 0.6029654741287231, 0.8367457389831543, -0.6872469186782837, 1.2192529439926147, -0.07066536694765091, 0.2641107141971588, 0.18698087334632874, 0.7840594053268433, 0.5463611483573914, -1.2148514986038208, -0.30071717500686646, -0.9163714051246643, -1.8444271087646484, -0.20011310279369354, 0.14766713976860046, 1.2226020097732544, -0.48284417390823364, -0.6474372744560242, -0.1302398294210434, 1.0071394443511963, -1.8600624799728394, 1.8693686723709106, 0.6038024425506592, 0.02398390881717205, 1.1552976369857788, 0.6913525462150574, 0.6158667206764221, 0.5876365303993225, -1.5378758907318115, 1.3801625967025757, -1.044963002204895, -0.3282926082611084, -0.27601906657218933, 1.4495877027511597, -0.7175120115280151, -0.8443800210952759, -0.48353880643844604, 0.33854377269744873, -0.5638027191162109, 1.238111972808838, -1.630008578300476, -0.8205034732818604, 0.09142009913921356, -1.6699693202972412, -0.0887613520026207, 0.1463635116815567, 1.444821834564209, 0.35153913497924805, -0.889127254486084, 2.3432302474975586, 1.9425355195999146, -0.12419896572828293, 1.8876205682754517, -0.6546730399131775, -0.584690511226654, -0.2192142903804779, 2.1740078926086426, -1.1561343669891357, -1.2182791233062744, 1.0996729135513306, 0.9203512668609619, 0.27072232961654663, -1.2639236450195312, -1.678247332572937, 0.2971307933330536, 1.5106970071792603, 0.5119870901107788, 1.1493581533432007, 0.23667468130588531, 0.09527434408664703, -1.004656434059143, 0.2914673984050751, -0.13959293067455292, 0.34328216314315796, -0.45443323254585266, -0.5332647562026978, 0.7217504382133484, 0.27679160237312317, 0.717190146446228, -1.267549991607666, 0.913341760635376, -0.3520072400569916, -1.0661120414733887, -0.48745518922805786, 0.4837760925292969, 1.5053712129592896, 1.5766057968139648, 1.340617299079895, -1.5624191761016846, -0.5279034972190857, -0.2676964998245239, 1.831020712852478, -1.520952582359314, -0.5103732943534851, 0.33347538113594055, 0.579261839389801, 0.47238579392433167, -0.6906167268753052, 0.5966417193412781, 1.0353761911392212, 0.07109416276216507, -1.062543272972107, -1.2649701833724976, -1.1550359725952148, 0.09987052530050278, 0.1569690853357315, -1.0756944417953491, -0.4033085107803345, -1.8451238870620728, -2.1604013442993164, 0.524070680141449, -0.8696748614311218, 0.08679481595754623, -1.5080221891403198, -0.702509343624115, 0.5839076638221741, -0.93473881483078, 0.7975203990936279, 0.6398343443870544, -0.19408144056797028, 1.679792046546936, -0.07443608343601227, 0.49682217836380005, -0.392447829246521, -0.21092885732650757, 0.34333112835884094, -0.4183201491832733, 0.5922380685806274, -1.5037301778793335, -0.450497031211853, 1.022317886352539, 0.09936264902353287, 1.328068494796753, 0.7979673147201538, -0.29058170318603516, -2.5793285369873047, -0.3571167290210724, -0.009931010194122791, 1.6461261510849, 0.6061572432518005, -0.27662229537963867, -0.4887474477291107, 0.5495216250419617, 0.7263056635856628, -1.3555370569229126, -1.7552059888839722, -0.7946807146072388, 0.4523801803588867, -1.2860485315322876, -0.8150809407234192, 1.0974498987197876, -0.9321447610855103, 0.4543059766292572, 0.4535185396671295, 0.4417148232460022, 1.779938817024231, -2.3348047733306885, 0.2902945876121521, 1.0831732749938965, 0.696202278137207, -0.3698868453502655, -0.47625845670700073, 0.6614988446235657, -0.03932923078536987, 0.933653712272644, -0.5396538972854614, -0.2479521781206131, -0.9587856531143188, 0.31947848200798035, 1.6403042078018188, 0.5883257985115051, -0.9618178606033325, 0.12810176610946655, 0.7049450874328613, -0.3036308288574219, 0.3351612985134125, 0.7807524800300598, -0.20639903843402863, 0.8836176991462708, 0.13789629936218262, 0.2844661772251129, -0.03517985716462135, -0.49508237838745117, -0.3378554880619049, 0.866753101348877, 0.0423295758664608, 0.8212983012199402, 0.7784945964813232, 0.8479787111282349, 1.5614163875579834, 1.206215262413025, 0.10225213319063187, 0.927232563495636, -0.6115687489509583, 2.5737760066986084, -0.3202834129333496, -1.9425746202468872, -0.6281400322914124, -0.464590847492218, -0.23899823427200317, -1.1611405611038208, -0.6419064998626709, -1.3260672092437744, -0.1454027146100998, 0.4613581597805023, 1.9555104970932007, 0.9564256072044373, -0.433769166469574, 0.522836446762085, 0.5620426535606384, -1.6569080352783203, -1.5579707622528076, 0.023451559245586395, -2.1353530883789062, 2.0707318782806396, 0.3723922371864319, 0.5453136563301086, 0.05464702472090721, 0.2538047432899475, -0.13763141632080078, 0.6807628870010376, 0.2113804668188095, -0.32348179817199707, -0.7216842770576477, 0.3508789539337158, -1.046270489692688, 1.226434588432312, -1.7749583721160889, 0.32692450284957886, 0.5644770860671997, 0.10215169936418533, 0.33854177594184875, -0.9339599609375, -0.026278574019670486, 0.7574989795684814, 0.48545026779174805, 0.22832445800304413, -0.8977545499801636, -0.5618495345115662, -1.2683244943618774, 2.1077687740325928, 0.7636716961860657, 1.1615498065948486, -2.1624414920806885, -0.9844074845314026, -0.5509109497070312, 0.5290729403495789, 0.20764413475990295, -0.6315010786056519, 1.292682409286499, 1.2912795543670654, 1.3318229913711548, -0.20409415662288666, -1.4118618965148926, -0.37259143590927124, 1.2522765398025513, 1.1159712076187134, 1.7561562061309814, -1.2442251443862915, -0.009934468194842339, -1.2479552030563354, -1.2906999588012695, 0.1943080574274063, -0.12916448712348938, -0.8051021695137024, -0.056682437658309937, 0.6087436676025391, -0.2644941210746765, -0.11482396721839905, -1.097501277923584, -1.6510484218597412, 0.20077243447303772, 1.0244815349578857, -0.7942622303962708, -0.2956012487411499, -0.11457809805870056, -1.1713297367095947, 0.8365135192871094, -0.47443756461143494, 0.9144616723060608, 2.1477813720703125, -0.4577523171901703, -1.2418230772018433, -1.5436235666275024, -0.817193865776062, -0.09638527780771255, 1.256735920906067, -0.8776392936706543, 0.3979979455471039, -0.6781551241874695, -0.8082117438316345, -0.024012308567762375, 0.8229308128356934, 1.1577116250991821, 0.16448664665222168, 1.58594810962677, 1.03529953956604, 0.6593875288963318, 0.38469991087913513, -0.05447527393698692, -1.0472307205200195, 1.6426488161087036, 0.21696950495243073, 0.6044728755950928, -1.1336621046066284, 0.09852644056081772, 0.07480733841657639, 0.6854264736175537, 0.28234240412712097, 0.7470940351486206, 0.6232008337974548, -1.6307694911956787, -0.2436329424381256, 1.8415677547454834, 0.5529549717903137, -0.3278525173664093, -1.1732162237167358, 0.3031913638114929, 0.04119335114955902, 0.3703983724117279, 0.09526193141937256, -0.5261039137840271, -0.13840922713279724, 2.867384672164917, -1.913848638534546, -1.3382220268249512, -1.011305570602417, 1.4932881593704224, -1.0406535863876343, -0.5520442724227905, -0.6239724159240723, 0.715086042881012, -0.12109432369470596, 1.7345426082611084, 0.10116619616746902, -1.81583571434021, 0.593413770198822, 0.11806793510913849, 0.21726322174072266, 0.26299184560775757, -1.8819894790649414, -0.16892381012439728, -0.2292792946100235, -0.6141343712806702, 0.6805010437965393, -0.1746576875448227, 0.7322977185249329, -0.47229185700416565, -2.5360636711120605, -0.5544314980506897, -0.9302642941474915, 0.454572856426239, -0.1215200275182724, 0.3756931722164154, -0.23904076218605042, 0.20481663942337036, -0.38501521944999695, 1.3764134645462036, -1.1221102476119995, -0.6191027164459229, -0.40532010793685913, 1.5757637023925781, -0.01771635375916958, 1.6858690977096558, 1.533639669418335, 0.2062802016735077, -0.3276091516017914, -0.48448529839515686, -0.5217694044113159, 0.744185745716095, -1.213545799255371, -1.1971486806869507, 1.5715481042861938, 0.1342376470565796, 0.450619101524353, -0.3218540847301483, -0.5431699752807617, 1.7937290668487549, -0.8359166383743286, 0.9698861241340637, -3.8086636066436768, -2.008192777633667, 0.464983731508255, 0.3160821199417114, -1.539260745048523, 0.6623609662055969, 1.3419184684753418, -1.9339802265167236, 0.3205845057964325, 1.0088261365890503, -0.07040175795555115, -1.1580877304077148, 1.8233543634414673, 1.343934416770935, -0.18026478588581085, -0.8192018866539001, 0.09533119946718216, 2.058180570602417, 1.1983152627944946, 1.4668854475021362, -0.14939987659454346, 1.6621609926223755, -0.0923047587275505, -2.0610809326171875, 1.1334612369537354, 0.9939407706260681, 0.18096493184566498, 2.1304314136505127, 0.033885929733514786, -1.4898210763931274, -0.1931990087032318, -1.4351640939712524, 0.7762638330459595, -1.2425647974014282, -0.12687209248542786, 0.7519720792770386, -2.415088176727295, 0.2908537983894348, 1.4398224353790283, 2.178178310394287, -1.3316845893859863, 0.5256356596946716, -1.287534475326538, 1.2353845834732056, 0.5550066232681274, -0.12122270464897156, -0.8719409108161926, -2.388619899749756, 0.5177003145217896, 2.1732072830200195, 1.0472122430801392, -0.6181548237800598, 0.6903041005134583, -0.10267741233110428, -0.36134281754493713, 0.8431956171989441, -0.5392850041389465, 0.2210792899131775, -1.6034029722213745, 1.683426856994629, 0.3936213552951813, 0.39897456765174866, -0.2875571846961975, 1.8275766372680664, -0.28621092438697815, -0.09760603308677673, 3.585826873779297, -0.48820894956588745, -0.8423492908477783, 1.6850868463516235, 0.4534027874469757, 0.9088565111160278, 0.5138489007949829, -0.4782737195491791, 0.3084871172904968, -1.0791915655136108, -0.6203473806381226, -1.1968497037887573, 0.6751473546028137, -1.260186791419983, 0.4982629418373108, -1.410475730895996, -0.6571620106697083, -0.9808777570724487, 1.5515735149383545, 0.6032838225364685, -2.4448981285095215, 2.2979679107666016, -0.4282935857772827, 1.9593923091888428, 0.2912151515483856, -2.596606492996216, 0.8012544512748718, -0.5200638771057129, 0.18040013313293457, -0.19757197797298431, 0.45057713985443115, -1.342616081237793, 0.4047524929046631, -0.2833317816257477, -0.702742338180542, -0.25569555163383484, -1.6450685262680054, 0.42643237113952637, -0.45124995708465576, -0.9056113362312317, 0.3503274619579315, 1.1899999380111694, -0.48528391122817993, 0.11848989129066467, -0.33377474546432495, 0.5653544068336487, -1.0590251684188843, 0.3828376531600952, 0.13609224557876587, 0.39153388142585754, 0.20402397215366364, -0.00023606274044141173, -2.178881883621216, -0.654687762260437, -0.1142694428563118, 0.19305214285850525, -0.25642964243888855, -0.24836422502994537, -0.21722795069217682, 0.5516858100891113, 1.8515156507492065, -0.4903978109359741, -1.4015529155731201, -2.114903211593628, -0.34681248664855957, -1.279188632965088, -0.4294966161251068, -1.2641438245773315, -0.3319914638996124, -0.11115933954715729, -0.42687371373176575, 0.774735689163208, 1.1124413013458252, -1.4081056118011475, 0.2326776534318924, 0.5334357023239136, -0.6423662304878235, -1.7324126958847046, -0.7811927199363708, 0.39010441303253174, 1.3363101482391357, -0.2749860882759094, 0.7960062026977539, -0.8005538582801819, 0.49976497888565063, 0.34387797117233276, 0.5964547395706177, -1.006718397140503, -0.4486329257488251, 0.5644047856330872, -0.005323314107954502, 1.6731480360031128, -1.9856370687484741, 1.3382569551467896, 1.2040009498596191, 1.1786551475524902, -1.2111133337020874, 0.21542499959468842, 1.3043484687805176, 1.2925183773040771, 0.16194984316825867, 0.5397354960441589, -0.18801285326480865, 0.06329726427793503, -2.6092028617858887, -0.5357838869094849, -0.09150028228759766, -0.7255094051361084, 0.33875682950019836, 0.37001800537109375, -2.094406843185425, -0.4920361042022705, -0.3939150869846344, 1.0928443670272827, -0.6577251553535461, -2.0055832862854004, 0.22729167342185974, -0.28076639771461487, -1.1962244510650635, -0.8223751783370972, 0.08180096000432968, 0.4892852008342743, -0.26944977045059204, -1.0554550886154175, 1.0464054346084595, 1.2212986946105957, -0.5507232546806335, -0.18334197998046875, 0.919975221157074, 0.11012884229421616, 2.0099377632141113, 1.006225824356079, 0.6879531741142273, 0.10843464732170105, -0.6581779718399048, 0.8004999160766602, -0.6858447790145874, -2.561358690261841, -1.508156180381775, -1.2375295162200928, 2.582618474960327, -0.48910123109817505, -1.3045777082443237, 2.0205953121185303, -0.7600414752960205, 0.9402212500572205, -0.7011981010437012, 0.15262672305107117, 0.12682777643203735, 0.0015222260262817144, 0.6262742280960083, 1.8885223865509033, -0.7373199462890625, -0.981472373008728, 0.24876238405704498, 1.2631876468658447, 0.8524923920631409, 1.3409764766693115, -0.6267359852790833, -1.156177043914795, 1.4271003007888794, -0.4631856083869934, -0.8241167068481445, 1.4750182628631592, -1.7864782810211182, -1.5424078702926636, -0.159047469496727, 0.7247254848480225, 0.6596230864524841, -0.9825888872146606, 0.00040534252184443176, 0.09639150649309158, 1.262977957725525, 0.0014277093578130007, -1.417852759361267, 0.4381482005119324, -0.08148638904094696, -0.6847301125526428, -0.2705153524875641, 0.19748255610466003, -1.0371605157852173, 0.7707954049110413, -0.9386774301528931, 0.28529566526412964, -1.7656499147415161, 0.5944854617118835, -1.0359326601028442, 0.2397935837507248, 1.3012886047363281, -1.3036444187164307, -0.6553264856338501, -1.3287733793258667, -0.8459720015525818, 0.4894387722015381, -0.7230714559555054, -0.6367509365081787, 1.3917311429977417, 0.8483863472938538, 1.2109730243682861, -0.37727418541908264, 1.5482364892959595, 0.8501381874084473, -0.9437279105186462, -0.72249835729599, -0.2380443811416626, -0.3472420275211334, 0.3980841338634491, -0.20883449912071228, 1.8784428834915161, 0.669530987739563, 0.3931088447570801, -0.6181380748748779, -0.24733655154705048, 0.1475762128829956, -0.24123139679431915, 1.8990111351013184, -0.9534431099891663, 1.4214617013931274, -0.6228209733963013, 0.21911419928073883, -0.8378612995147705, -0.3799097239971161, -1.3944910764694214, -0.9451960325241089, 1.3270163536071777, -0.35884812474250793, -0.06769023090600967, 1.0489898920059204, -0.4672633707523346, -1.1663751602172852, -0.2103421837091446, 1.7120604515075684, -0.6044716835021973, 0.11358693242073059, 0.974017858505249, 0.7175988554954529, -0.4526219964027405, 0.1096540316939354, -0.302501380443573, 0.16784092783927917, -1.0136454105377197, 0.7150171995162964, -0.17387796938419342, -0.6957963705062866, -0.7196743488311768, 0.6347883343696594, 0.3117932975292206, 1.5951259136199951, 0.42431798577308655, 0.12578806281089783, 1.0865811109542847, 1.3640635013580322, 1.4981372356414795, 0.6336464285850525, -0.9718031883239746, 1.2571747303009033, -0.6085721254348755, 0.4904060661792755, 1.1702316999435425, -0.8562440276145935, 0.9494125843048096, 1.5021626949310303, -0.40324532985687256, 0.49216556549072266, -0.28694581985473633, -1.788676381111145, -1.0355098247528076, 0.6250664591789246, -0.04931548237800598, 0.5535362958908081, -0.07709124684333801, 0.3947378098964691, -0.7645958662033081, -1.0830111503601074, -0.1395440399646759, -0.07073463499546051, 1.9234815835952759, -0.5102875828742981, -1.4880375862121582, -0.482799768447876, 0.02058013714849949, 0.4787280857563019, -0.3016511797904968, 0.11117285490036011, 0.6547031998634338, 1.8152194023132324, -1.2316347360610962, 0.917996883392334, 0.013574252836406231, 1.1828699111938477, 0.5860446691513062, -1.071664810180664, -0.49300912022590637, 1.1198915243148804, 2.913353443145752, 0.313427597284317, 1.865246057510376, -0.4242466390132904, -0.8966554999351501, -1.5197733640670776, 1.1669930219650269, 0.9672468304634094, -2.402759075164795, 0.11829080432653427, -0.2020290046930313, 0.6413363218307495, 0.9538189768791199, 0.6948491930961609, 0.9161077737808228, -1.5348752737045288, 0.8625762462615967, 0.9568737745285034, -0.17060621082782745, 1.734948992729187, -0.8929805159568787, 0.42401963472366333, 0.17451126873493195, -0.20257768034934998, -0.5580076575279236, -0.9123759269714355, -0.22403942048549652, -0.5557135939598083, -0.4816040098667145, 0.37586236000061035, -0.5029045343399048, -0.13273213803768158, -0.32825034856796265, 0.9004512429237366, 0.4735018014907837, -0.2389698475599289, -1.8632892370224, 0.0790402963757515, 1.112295389175415, 2.3124897480010986, -1.1589887142181396, 0.09689871221780777, -0.15069785714149475, 0.3271300494670868, -1.0532426834106445, -2.604687213897705, 1.2242499589920044, 1.1410086154937744, 0.25511980056762695, -0.7557850480079651, 1.270114779472351, -1.0064244270324707, 0.04979193955659866, -0.6122106909751892, -0.6605513095855713, -0.1874532401561737, -0.12558533251285553, -0.24474374949932098, -0.20474529266357422, 0.0028224466368556023, -0.33142387866973877, -0.2762407064437866, 0.03244875743985176, -0.7496687173843384, -0.01552653405815363, -0.13522455096244812, 0.8153500556945801, -0.1674797534942627, -1.4566216468811035, -0.01501296553760767, 1.0963648557662964, -0.28057023882865906, 0.12747828662395477, -0.9352880120277405, -0.7930359244346619, -0.6926189661026001, 0.18003389239311218, 1.5312566757202148, -0.6127782464027405, -0.7107869386672974, 1.3132884502410889, -0.36570173501968384, 0.7513702511787415, 1.349034309387207, -0.7340747117996216, -0.3081636130809784, 0.79465252161026, -1.1639938354492188, 0.5316853523254395, -0.6916162371635437, 0.8498539328575134, 1.0829699039459229, 1.0598318576812744, 0.5451200008392334, 0.05846302583813667, -0.3390346169471741, 0.39267468452453613, -1.331743836402893, 1.295811414718628, -1.3348770141601562, -0.941599428653717, -0.9752289056777954, -0.20887504518032074, 0.083188496530056, -1.0618518590927124, 0.3304333984851837, -0.3589833974838257, -1.299765706062317, 1.3913962841033936, 0.6428854465484619, -0.8171914219856262, -0.24322792887687683, -0.934899628162384, -0.2535635530948639, 2.2433605194091797, 1.1371839046478271, -0.3359513282775879, -0.003408940741792321, 0.4388265907764435, 1.0414824485778809, -0.01261176448315382, -1.131628394126892, -0.2960793375968933, -0.23095789551734924, -0.1804986149072647, -0.7585195302963257, -0.4887505769729614, 1.013426423072815, 1.0250608921051025, 1.4527159929275513, -0.44713571667671204, -0.6978197693824768, -0.0730501264333725, 0.730572521686554, 1.3893758058547974, -0.6199505925178528, 0.09802458435297012, -0.31061142683029175, -0.7183239459991455, -3.3333990573883057, 0.6581125855445862, -0.7674590945243835, -0.2365717887878418, -0.19292117655277252, 0.15025247633457184, -0.684627115726471, 0.6852529048919678, 0.6353568434715271, -0.674978494644165, 0.4684523940086365, -0.24706301093101501, 0.7754725217819214, -1.1107991933822632, 0.7255873680114746, -1.3706220388412476, -1.0052400827407837, -0.6845162510871887, 0.22177761793136597, -0.12765182554721832, 0.17583878338336945, 0.4001908302307129, 1.6748343706130981, 0.25037881731987, 1.5814611911773682, 0.009603405371308327, -1.1522190570831299, 1.8689119815826416, 0.09958745539188385, 0.2833390533924103, -0.47324126958847046, -0.1334136426448822, -0.6410586833953857, 0.06948978453874588, -1.5675950050354004, 0.14001579582691193, -0.2634921371936798, -0.5082928538322449, 1.5238454341888428, -0.24704709649085999, -0.7374048829078674, 1.148390769958496, 1.0730897188186646, -0.008295120671391487, 1.6630750894546509, 0.20665523409843445, -0.0594400092959404, 2.424356460571289, -0.5948176383972168, 1.4263032674789429, -1.4966278076171875, -0.8842142820358276, 0.02725045382976532, 0.40991538763046265, -0.5303919315338135, -0.5386804938316345, 0.989403486251831, 0.20506848394870758, 0.2578057646751404, 0.7104113698005676, 0.10846208035945892, -0.29155659675598145, 0.9207481145858765, -1.0253347158432007, 0.11160878092050552, -1.7094922065734863, -2.4892940521240234, -0.3524981141090393, -0.3645057678222656, -0.7574055194854736, -0.9525278806686401, 0.09215589612722397, 0.6119401454925537, -0.8575329184532166, 1.9632749557495117, -0.8393318057060242, -0.36885538697242737, -0.5499789714813232, 0.10299735516309738, -0.9744760990142822, 1.3534704446792603, 1.7445212602615356, 0.6237545013427734, 2.9702162742614746, 0.3875653147697449, 0.45725852251052856, -1.3886865377426147, -0.9976429343223572, -0.6984962224960327, 0.47811105847358704, -0.10148625820875168, -0.23925255239009857, 0.10317666083574295, 0.5516418218612671, 0.3595316410064697, 0.09096982330083847, -1.490219235420227, -1.2936381101608276, 1.2199510335922241, 1.9333686828613281, -0.025881750509142876, 0.11284536868333817, 0.1545257568359375, -1.071042537689209, -0.020565729588270187, -0.8473731875419617, -0.1569986641407013, 0.34129634499549866, -0.07717114686965942, 0.5859916806221008, -0.5141134262084961, 0.5608969330787659, -1.1862030029296875, -1.5605493783950806, -0.13834993541240692, -0.38904276490211487, 0.0474376305937767, 0.17472849786281586, -0.5638571977615356, -0.415218710899353, -0.638927698135376, -0.8372513651847839, -0.7592064142227173, -1.1454654932022095, -0.8678951859474182, 0.08386576175689697, -1.8620065450668335, -1.6862926483154297, -0.7966267466545105, 0.015250113792717457, 0.8421606421470642, -0.3505856990814209, -0.910775363445282, 0.1592579483985901, 1.2824339866638184, -0.3461230993270874, 0.7163034677505493, 1.0368297100067139, -0.3290518522262573, -0.4387196898460388, -0.13553351163864136, 0.33985060453414917, -1.0943787097930908, 1.4379793405532837, -0.01564466953277588, 0.15456144511699677, 0.644212543964386, -0.050081223249435425, -0.643003523349762, -0.7378391623497009, -0.579990029335022, -1.1723512411117554, -0.4834814965724945, 0.16860529780387878, 0.10266391187906265, -0.564176619052887, 0.22778232395648956, -0.46895983815193176, 0.726324200630188, 1.5050408840179443, -0.9051281213760376, 0.8940613865852356, 0.35069918632507324, -0.1724519580602646, 0.5784409046173096, -1.0043916702270508, 0.8347668051719666, 1.813252329826355, 1.1599531173706055, 0.35873880982398987, 1.7752013206481934, 2.111980438232422, -2.0243916511535645, 0.0477922186255455, -0.3763636648654938, 0.4480549693107605, 0.4766969680786133, -0.7972798943519592, 0.31035831570625305, -0.47739502787590027, 0.3027113974094391, -1.0980976819992065, -1.9231534004211426, -0.7220457196235657, 1.691001057624817, -0.5264744758605957, -0.31712520122528076, 1.3371105194091797, -0.594802975654602, 0.2198549062013626, -1.5281544923782349, 1.4527486562728882, -0.242500439286232, -0.29153627157211304, 1.4552973508834839, 0.07880546152591705, -0.14320108294487, -1.1118813753128052, -0.9098613858222961, -0.6458548903465271, 0.4342040419578552, -0.20713096857070923, -0.09965316206216812, -0.04476102069020271, -0.515144944190979, -0.2573888599872589, -1.486158013343811, 0.316303014755249, 1.0373369455337524, 0.38137295842170715, -2.439389944076538, -0.5207028388977051, -1.0190545320510864, -0.03921826183795929, 0.3007766306400299, 1.7749013900756836, -0.2965472340583801, -0.310838520526886, -0.42753365635871887, 1.414017677307129, 1.4003214836120605, 0.5049219727516174, -0.1277041733264923, -0.12160620093345642, -1.4290186166763306, -0.020128563046455383, 0.5863465666770935, -1.5543760061264038, 1.1118426322937012, 0.5313984155654907, -2.3339297771453857, -0.16111011803150177, -1.1203510761260986, -0.808623731136322, -0.27191948890686035, -0.10148708522319794, -0.1866849809885025, 0.7182176113128662, -0.9442884922027588, -1.67203688621521, 0.752234935760498, -1.617750644683838, 0.7332345843315125, 1.0138766765594482, 0.6808452606201172, 0.011526353657245636, -1.4804518222808838, 0.484123557806015, 1.2355016469955444, 0.8357691168785095, 1.562574028968811, 0.47826874256134033, 2.3053441047668457, -1.220429539680481, -1.0789637565612793, -0.7741320133209229, 0.5000130534172058, -0.527168333530426, -0.9956972002983093, 0.32769888639450073, -0.2872711420059204, 1.6656887531280518, -1.9741038084030151, -0.7210201025009155, -0.6463971138000488, -0.9572387933731079, -0.22678571939468384, -1.066659688949585, 0.037749484181404114, -0.47072941064834595, -1.1990336179733276, 0.2169618010520935, 0.9201063513755798, 0.3464702367782593, 0.1813536435365677, 0.4912888705730438, -0.0133588295429945, -0.036106936633586884, -3.187786817550659, 0.8307927846908569, 0.42655816674232483, 0.23516978323459625, 0.48490896821022034, -1.9143296480178833, 1.1583948135375977, -2.185270071029663, -1.4456164836883545, 1.6410548686981201, 0.3363293707370758, -1.7196084260940552, 0.16997571289539337, -0.027717074379324913, -0.1480424553155899, -0.024689925834536552, -2.2574973106384277, -1.1573735475540161, -1.444685697555542, -1.2727762460708618, -0.7488813996315002, 2.2515320777893066, -0.6260969638824463, 0.6165204644203186, 0.6121826767921448, -0.22801005840301514, 1.1769450902938843, 0.1992611289024353, -0.726578414440155, -0.634712815284729, 1.0024394989013672, 0.9582006335258484, -0.859830379486084, 0.02030950039625168, 0.7747251987457275, -0.9665512442588806, 1.222649335861206, -0.8173168897628784, -0.995667040348053, 0.9056174755096436, -0.5679169297218323, 1.6583906412124634, -0.9803639054298401, 1.5048996210098267, 1.7706172466278076, 0.11808713525533676, -1.3912248611450195, -0.27714669704437256, -1.1205195188522339, 0.29269999265670776, 0.24727530777454376, -0.060336291790008545, -1.2371318340301514, 1.3503135442733765, 0.3303054869174957, 1.8142234086990356, 1.6562660932540894, 0.48657745122909546, -0.2723056972026825, 0.20925448834896088, 0.6924271583557129, -0.7677631378173828, -0.5865437984466553, -1.5579239130020142, 1.27581787109375, 0.38283106684684753, 0.2808479964733124, -2.9188785552978516, -0.6662977933883667, 0.9191856384277344, -0.24141749739646912, 1.401219129562378, -2.0589592456817627, 0.9301231503486633, 0.24338442087173462, 0.9784338474273682, 1.7509629726409912, 1.4238606691360474, -0.0949196070432663, -0.2105400115251541, -1.030394196510315, 0.48738405108451843, -0.15963035821914673, -0.11402064561843872, 0.032393429428339005, 0.30755746364593506, -0.8590268492698669, 1.0855612754821777, -0.06606201082468033, -0.33534738421440125, 0.5065895915031433, 0.5799367427825928, -1.4666029214859009, 1.2819827795028687, 0.34972330927848816, -1.2524763345718384, 0.05111318454146385, -0.9415059089660645, 1.1615040302276611, 1.3789441585540771, -0.5767348408699036, -0.9889910221099854, -0.7962681651115417, -1.2816864252090454, 0.07210789620876312, -1.7837097644805908, -0.14489662647247314, -1.19416081905365, -0.16911442577838898, 0.054958563297986984, -0.09947721660137177, -0.5141138434410095, 2.0509836673736572, 0.12487421929836273, -0.3512837588787079, -0.15867659449577332, 0.7562960982322693, 1.9284919500350952, -0.7983083724975586, 0.3514593541622162, -0.5793306231498718, -0.16500161588191986, 1.6184895038604736, 0.9268848299980164, -1.0046652555465698, 0.6772757172584534, -0.13268302381038666, 0.24676446616649628, 0.7272411584854126, -0.9416292309761047, 0.2667982876300812, 1.5232794284820557, -0.09967371076345444, 1.7675223350524902, 0.8820798993110657, -1.0636998414993286, -0.9965896606445312, -0.11761543899774551, -0.7255882620811462, 0.024263447150588036, -0.37940263748168945, 1.7168232202529907, 1.7434927225112915, -0.556124746799469, -0.7571262121200562, 1.649420142173767, 0.295236736536026, 0.9575192928314209, 0.9068781137466431, -1.0832324028015137, 0.22486259043216705, 0.11891684681177139, -1.4460912942886353, -0.8101760745048523, 0.2951512038707733, 0.3944167196750641, -0.16478261351585388, -1.3263493776321411, 0.5525830388069153, 2.53910756111145, 0.12660804390907288, -0.14034061133861542, -0.431959867477417, -1.0542834997177124, 0.1395823061466217, -1.6533691883087158, -1.7787235975265503, -1.707281470298767, 1.0838596820831299, -0.2727741301059723, 0.8836604952812195, -2.0362398624420166, -1.2594813108444214, -0.5086968541145325, -1.2218568325042725, -0.936426043510437, 0.36086103320121765, -0.6884899735450745, 0.19163401424884796, 1.07249116897583, 0.8563118577003479, 1.277488112449646, -1.0852265357971191, 0.5929372310638428, 1.8202457427978516, 1.6061311960220337, 0.7651886940002441, -0.016745634377002716, 0.705217719078064, -0.18091179430484772, 0.4400164484977722, 1.5104438066482544, 0.026631362736225128, -1.1509593725204468, 1.2261606454849243, 1.2340425252914429, 0.45467454195022583, -0.11600513756275177, 0.030645903199911118, -1.36367666721344, -0.6073353290557861, -1.9060126543045044, -1.0036581754684448, -1.5010101795196533, -1.7420861721038818, -1.7865976095199585, -0.1283433437347412, 0.23673950135707855, -1.0495765209197998, 0.4568277895450592, 0.5043716430664062, 0.12277455627918243, -1.1398751735687256, 1.0130674839019775, 0.22572821378707886, -0.46221187710762024, -0.22083882987499237, 0.13234291970729828, 1.5472307205200195, -0.49199074506759644, 1.3361520767211914, -1.3729970455169678, 1.1581792831420898, -1.2272504568099976, -0.6253589987754822, 0.46513622999191284, 0.3656788170337677, -0.3852510452270508, 1.0846656560897827, -0.5656628012657166, -1.0860792398452759, 0.7917559742927551, -1.6025336980819702, -0.22201970219612122, -0.7217912077903748, -2.216442346572876, 0.4578009247779846, -0.09253554046154022, -1.9279959201812744, 0.4186357259750366, 0.006562521681189537, -0.6694725155830383, -1.3801318407058716, 0.7588515281677246, 1.247793436050415, 0.3033786416053772, -0.4758046269416809, -0.9122676253318787, 0.15754766762256622, 0.16050896048545837, 1.1208547353744507, 0.14356684684753418, 1.6027432680130005, -0.6097273230552673, 0.5760367512702942, 0.2534213364124298, 1.1277140378952026, -0.7236465215682983, 1.457576870918274, 0.31502965092658997, -0.4368171989917755, -0.6223031878471375, 1.9709396362304688, 0.5569257736206055, -1.5084882974624634, 0.143728569149971, -0.6861610412597656, -1.606592059135437, 0.44418373703956604, -0.7501295804977417, -0.5197678208351135, -1.8498163223266602, 1.4673690795898438, 2.28426194190979, -0.9221250414848328, 0.24870207905769348, 1.403505802154541, 0.3916318416595459, -0.5537561178207397, -1.6652361154556274, 0.971942126750946, -0.5503974556922913, -0.83207768201828, -0.5636944770812988, 0.22990602254867554, 0.8770689368247986, -0.2988189160823822, 1.8604869842529297, 0.20524565875530243, -1.2859477996826172, 0.6881566047668457, 0.32678142189979553, -1.163940191268921, 0.11980514228343964, 2.061870813369751, -1.1579997539520264, 0.47956302762031555, -1.134318232536316, 1.2530544996261597, -1.3211861848831177, 0.6054027080535889, -0.09943652898073196, -1.0256356000900269, 0.4770936369895935, -0.377515971660614, -0.003910287283360958, -0.7989808917045593, 0.9118232727050781, -1.0945312976837158, 0.6480805277824402, -2.201885938644409, -1.1343967914581299, 0.4758298695087433, 0.8553482890129089, -1.940200686454773, -1.4750275611877441, -0.8779138326644897, -1.604297161102295, -1.3494045734405518, -1.1815154552459717, 1.7991838455200195, -0.9570057988166809, 0.42700374126434326, -0.7446187138557434, -0.9882619976997375, -1.8840409517288208, 0.7865239977836609, 1.61600923538208, -0.4342361390590668, -1.1507647037506104, 0.653404712677002, -0.10890951007604599, 0.6668485999107361, 1.4714446067810059, 0.21721981465816498, 0.8103966116905212, 0.04949566721916199, 0.720338761806488, 0.6634928584098816, -1.368119716644287, -1.0096856355667114, 1.2756948471069336, 0.4828624427318573, 0.7474512457847595, 1.7557376623153687, 1.5138227939605713, -0.4701913297176361, -0.5785232782363892, -0.17213474214076996, -2.803422212600708, -0.30589601397514343, 1.045994520187378, 0.5593043565750122, -0.8996959328651428, 0.697895884513855, 2.169387102127075, 1.2194721698760986, -0.34661492705345154, 0.054197534918785095, -0.06193196773529053, -0.5747779011726379, 1.045366644859314, 0.2335406094789505, 0.5914279818534851, 0.49646833539009094, 0.2732243239879608, 2.1153581142425537, -0.4035482108592987, -0.06452430039644241, 0.5971193313598633, 0.7901395559310913, -1.1159636974334717, -0.06409570574760437, 0.878897488117218, -0.8430097103118896, -0.22339168190956116, -0.9442647099494934, -1.9672318696975708, 1.0077457427978516, 0.835328996181488, 0.1542191505432129, -0.43839746713638306, -0.2535090148448944, -0.554658830165863, 0.1894284188747406, 0.22396618127822876, 1.2351748943328857, -0.03456360101699829, -0.3858075737953186, -0.3599463403224945, 0.5862454175949097, 1.916846513748169, 1.782415747642517, -0.6854515671730042, -0.8147256374359131, 0.453806608915329, -0.24080204963684082, 1.0132871866226196, 0.7692386507987976, -0.7290489673614502, 0.658806324005127, -1.0663056373596191, -0.9434237480163574, -0.2050473839044571, 1.6068916320800781, -1.5096007585525513, 0.04227185621857643, -0.22539053857326508, -0.23654012382030487, -0.34506916999816895, 0.2130681425333023, -0.6229602694511414, -0.43135932087898254, -0.8984630107879639, 1.1389988660812378, -0.2251524031162262, -0.04995490238070488, -1.6252559423446655, -1.6148172616958618, -0.05732358247041702, -1.3360629081726074, 2.4869635105133057, 0.03297639265656471, 0.9668936133384705, 0.5203155875205994, -1.2813299894332886, 0.04575103521347046, 0.6897023916244507, -0.44391995668411255, -0.9924116730690002, -0.18613675236701965, -1.5811779499053955, 2.09081768989563, -0.822447657585144, -0.6979201436042786, -1.1174745559692383, 0.683819055557251, 0.945041835308075, 0.6031186580657959, 0.7989274263381958, 0.06607592850923538, -1.6397677659988403, 0.25311437249183655, -1.110422134399414, -0.8558861613273621, 0.03786671534180641, -0.6229466199874878, -0.09341899305582047, 0.8497776389122009, -1.315443754196167, -0.19533796608448029, -1.1922117471694946, 0.6906496286392212, 1.1727937459945679, -0.5781371593475342, -1.247245192527771, 0.7404548525810242, -0.38807839155197144, -0.21346546709537506, -0.6567530632019043, 0.5524293184280396, 0.16188722848892212, 1.2029812335968018, -0.3129749894142151, -0.8127081394195557, -0.3445190489292145, -0.9561517834663391, 0.22511819005012512, -1.222771167755127, 0.8174541592597961, -0.4519839584827423, -0.11152340471744537, 0.45212242007255554, -0.667184591293335, 0.5889460444450378, -0.46225517988204956, -0.3643110394477844, -0.45068269968032837, -0.9940283298492432, -0.4543386697769165, 0.40961262583732605, -0.5126320719718933, -0.4862525463104248, 1.5263831615447998, 1.1153920888900757, 1.2040334939956665, 0.9791149497032166, -0.2665835916996002, -0.07668087631464005, 0.21526530385017395, 1.3764537572860718, 0.44328153133392334, -0.2612592875957489, 0.13094119727611542, -0.3661476671695709, -0.17255087196826935, 0.5343234539031982, -1.3910479545593262, 0.605247974395752, 1.1447826623916626, -2.3475329875946045, -0.03848152235150337, 3.0434205532073975, 0.39810293912887573, 0.8126767873764038, -0.19940617680549622, -0.7770474553108215, 0.6903774738311768, -0.9443444609642029, -0.06970786303281784, -0.6182120442390442, 0.984493613243103, 1.0165709257125854, 0.916273832321167, 0.9133129715919495, -1.1517903804779053, -0.18697261810302734, -1.8622708320617676, 0.7338275909423828, 0.27574723958969116, -1.1164300441741943, -1.2600317001342773, 0.03309224918484688, -1.1139732599258423, 2.0589520931243896, 1.0514739751815796, -0.37865856289863586, -0.7053806185722351, 0.9929459691047668, -2.040534019470215, 1.6700773239135742, -0.9310763478279114, -0.43131548166275024, 0.7493193745613098, -0.20910798013210297, 0.04793458804488182, -0.48356202244758606, -0.17657479643821716, 0.7019301056861877, 0.7009309530258179, -1.5443087816238403, 1.215738296508789, 1.2503292560577393, 1.6419328451156616, 0.09346721321344376, -0.41102319955825806, -0.054668135941028595, -0.16757595539093018, 0.33007562160491943, -0.9637336730957031, -1.9860788583755493, 0.28243550658226013, 0.16794057190418243, 0.5946502089500427, -0.3406098783016205, -1.0074712038040161, -0.06558398902416229, -0.4123137891292572, 0.20638464391231537, 1.32008957862854, -0.38885191082954407, 0.02593495137989521, 0.9723391532897949, -0.14536459743976593, 0.6245831251144409], [0.9811461567878723, -0.9648062586784363, 0.5382890701293945, -0.677547812461853, -1.2830908298492432, 0.2136053442955017, -2.360654592514038, 0.656663179397583, -1.3451019525527954, 0.8789694309234619, 0.1673882007598877, -0.5208109021186829, 0.633215606212616, -0.6357025504112244, 2.638805866241455, -0.3351408839225769, 0.2898799180984497, 0.5431475639343262, 0.21262003481388092, -0.5923975706100464, -0.6536708474159241, 1.110032558441162, -0.9095306992530823, 0.8087185025215149, -1.9913785457611084, -0.369666188955307, -1.9409679174423218, -2.598144292831421, 0.4745587706565857, 0.34315383434295654, -1.6750613451004028, 0.6891628503799438, 0.39483413100242615, -1.6249372959136963, 1.2130661010742188, -1.029429316520691, 1.6592053174972534, 0.3938997685909271, 0.49106666445732117, -0.15697550773620605, 1.6269547939300537, -0.3435351550579071, -0.3881143033504486, -0.11196742951869965, -0.9502397179603577, -1.0462305545806885, 0.11112295091152191, -0.1239176094532013, 0.48339545726776123, 1.141937255859375, -0.23325560986995697, 0.26424744725227356, -0.37001466751098633, -0.4104978144168854, 0.6155555248260498, -1.170088768005371, 0.1967899203300476, 0.16770166158676147, 0.07178094238042831, -0.41613632440567017, -1.5980079174041748, -1.5782886743545532, 0.03695949539542198, 0.2090425044298172, -0.03519296646118164, 1.5646072626113892, 0.3560132384300232, 2.1581878662109375, 1.7145572900772095, -0.5194740891456604, -0.30198025703430176, 2.6603188514709473, -1.3696163892745972, -0.6984050869941711, -0.4138679802417755, -0.843802809715271, -0.22480152547359467, 0.019876405596733093, -1.497530460357666, -0.014448339119553566, 0.178681880235672, 0.5205232501029968, 0.5745465755462646, 1.0635758638381958, 0.7757693529129028, -0.6685522198677063, 0.5135119557380676, 2.065814971923828, 0.7003605961799622, -0.6294448971748352, -0.43063971400260925, -0.8257330656051636, -0.3597636818885803, -0.840861976146698, -1.1119567155838013, -0.7063302397727966, -1.67406165599823, 0.2851148545742035, 1.1329281330108643, 1.525956392288208, -0.5608044862747192, 1.972542405128479, -1.1181323528289795, 1.2650688886642456, -0.7249823212623596, 0.26394787430763245, 0.0403219535946846, -0.6296319365501404, -0.30169034004211426, 0.551441490650177, -2.635512351989746, -0.3692958950996399, -0.2862907648086548, 0.4452070891857147, -0.3549639880657196, -0.1742180436849594, -0.019676491618156433, 1.0683149099349976, -0.028194662183523178, 0.23583242297172546, 0.07059678435325623, 1.9619964361190796, 0.8443505764007568, 0.017779305577278137, -1.0171713829040527, 0.4809412956237793, 2.3075194358825684, -0.4828687310218811, -0.7294420003890991, -0.6113547086715698, 1.2446156740188599, 2.331717014312744, 1.0161229372024536, 0.521358072757721, 0.026127882301807404, 0.5617991089820862, 0.1714363992214203, -1.0906201601028442, 0.8861850500106812, 0.12086209654808044, 1.14629065990448, 1.41220223903656, 0.007087105885148048, -0.9495665431022644, 1.323105812072754, 0.1644008457660675, 0.4622780382633209, -2.1499557495117188, 0.5132845640182495, 0.2877636253833771, 1.251328945159912, 1.1190143823623657, -1.252741813659668, -0.537055492401123, 0.6236209869384766, -0.19190062582492828, -0.2843480706214905, 2.291822910308838, 2.6305108070373535, 1.7340031862258911, -0.8125677108764648, -0.3248993456363678, -1.8746322393417358, 0.41233980655670166, -0.17597290873527527, 0.21512150764465332, -1.4118977785110474, -1.2550102472305298, -0.8830357193946838, 0.6023883819580078, 0.7553927302360535, -0.490405410528183, -0.5863017439842224, -0.11783670634031296, 0.10105309635400772, 2.003936767578125, -0.8959711194038391, -0.47471708059310913, -0.1044904813170433, 0.04156581312417984, 0.3669959008693695, -0.2836263179779053, -1.2690980434417725, -0.8329979777336121, 0.4839256703853607, -0.24008245766162872, -0.4115917682647705, -0.13184848427772522, -0.6715887188911438, 1.5454844236373901, -0.8483806848526001, -1.40669846534729, -2.5633156299591064, -1.1366804838180542, 0.6133986711502075, -0.888793408870697, -0.4431757926940918, 1.223154067993164, -0.12916801869869232, 0.44974595308303833, 0.5272565484046936, -0.4570334255695343, 0.41686058044433594, -1.3018195629119873, -1.158075213432312, 1.8242079019546509, -0.6723418235778809, 1.2622061967849731, 0.6601359844207764, -1.175369381904602, 0.018824748694896698, -0.14944875240325928, -0.6323321461677551, -0.574781060218811, 0.531679093837738, -0.5603393316268921, 0.9755340218544006, -0.05782635882496834, 0.8115764856338501, -0.04427509382367134, -0.5503388047218323, -1.2086910009384155, 0.507239580154419, -2.6371102333068848, 0.1373177319765091, 1.6168941259384155, -0.8497002124786377, 1.411339282989502, 0.44505271315574646, -1.7614192962646484, -3.4325215816497803, 2.0963799953460693, -0.33638808131217957, -0.9402511715888977, 0.9109178781509399, 0.33171918988227844, 0.5571764707565308, 0.21323946118354797, 1.537491798400879, -0.8857259154319763, 1.737288475036621, -0.26107218861579895, 2.42463755607605, 1.425607681274414, -0.7198894619941711, -0.5840480327606201, -0.2990352511405945, -0.34331029653549194, 0.48496782779693604, -0.5657998323440552, -0.4973281919956207, 0.5951301455497742, 0.13849298655986786, -0.32807499170303345, 0.8296424150466919, 1.958430528640747, 0.618577778339386, -2.3049416542053223, 0.814171314239502, -0.5476232171058655, 0.3841121196746826, -0.7795345187187195, 0.06039970740675926, 0.4804786443710327, 0.7437214851379395, -0.16237765550613403, 0.6555540561676025, -0.053544625639915466, -0.0802217349410057, -0.6480543613433838, -1.6283289194107056, -0.6406806111335754, 0.36933907866477966, -0.22264325618743896, -1.929248332977295, -1.2515385150909424, 1.9352672100067139, -1.756795048713684, -1.5336889028549194, 0.7528367042541504, -0.6107024550437927, 0.6418712735176086, 0.407647967338562, 1.1616995334625244, -0.41579878330230713, 0.4605903923511505, 0.08396957069635391, 0.03900105133652687, 1.4725533723831177, 0.6322447061538696, 0.6534271240234375, -0.20116634666919708, 0.19749632477760315, 1.9523314237594604, 1.4926021099090576, -0.4815923571586609, 0.8642063140869141, -2.3445050716400146, -0.9794037938117981, -0.3777692914009094, 0.9279014468193054, -1.3996449708938599, 0.7066558599472046, -1.475262999534607, 0.6716110110282898, 0.442800909280777, 0.2564544081687927, -1.1225121021270752, -0.25318941473960876, 1.3398311138153076, -1.473969578742981, 1.3020148277282715, 2.4365127086639404, 1.855802297592163, -1.0493433475494385, 0.9432471394538879, -1.1697818040847778, 0.7391439080238342, 0.12142106890678406, -0.501105785369873, -0.28810325264930725, -0.2978513240814209, -0.09712453186511993, 0.6487956047058105, -0.13471268117427826, 0.28498995304107666, 0.23987267911434174, -1.660470724105835, 1.4835485219955444, 0.7806585431098938, 1.3879872560501099, 0.6414170265197754, 0.23457999527454376, 0.9007160067558289, -0.37934938073158264, 0.23747536540031433, 2.0839896202087402, 2.599123477935791, 0.3021742105484009, 0.452818363904953, -0.07105283439159393, -0.27785512804985046, 0.6445685029029846, 0.7448015809059143, 0.5783746242523193, -0.6353228688240051, -0.7789162397384644, -0.7273075580596924, -0.33450329303741455, 1.582730770111084, 0.37937062978744507, 0.31781917810440063, -0.08817330747842789, 0.4608851969242096, 0.5917221307754517, -0.6349624395370483, 0.15102319419384003, 0.4262031018733978, 0.3255878686904907, 0.3551994860172272, -0.0947573110461235, 0.4241291880607605, -1.7095026969909668, 0.3972886800765991, 0.7979140281677246, -0.8477243781089783, -0.07811855524778366, -0.1645481437444687, 1.1400904655456543, -1.2592881830642e-05, -0.39948827028274536, -1.0112779140472412, 0.8260934948921204, -1.2712721824645996, -2.4520411491394043, -0.22507202625274658, 0.093851737678051, -0.0046567153185606, -0.6443497538566589, 0.14103379845619202, 0.5759131908416748, 0.015106015838682652, 1.3913570642471313, -0.958722710609436, 0.01576686091721058, -0.9762833118438721, 1.296722650527954, -0.6862571239471436, -0.14792384207248688, 0.7570928335189819, -0.9498258233070374, -0.013688319362699986, -0.4625297486782074, 1.2163236141204834, -0.16921043395996094, 0.6085711717605591, -0.7314308881759644, 0.9969421625137329, 1.270622968673706, 0.5903156399726868, 1.197752594947815, -0.42284879088401794, 0.007254408672451973, 0.07141407579183578, -1.1542607545852661, -0.11549729108810425, 2.266227960586548, -0.8796970844268799, -0.23292943835258484, 0.4470374584197998, 1.9047341346740723, -0.6113829016685486, -0.14557121694087982, 0.5107647776603699, -1.7835191488265991, 1.4688034057617188, -0.6956354975700378, -1.1686712503433228, -0.5930685997009277, -1.4458413124084473, -0.19464297592639923, 0.3442600965499878, 1.3452280759811401, -0.39710861444473267, -0.8598288297653198, -0.22089578211307526, 0.7809937596321106, 0.946194589138031, -1.1480761766433716, 0.59783536195755, 0.2306925654411316, -0.6288811564445496, -0.7508731484413147, -1.335161566734314, 0.5840030908584595, 0.3563016951084137, 0.416559636592865, -2.837251663208008, 0.7853622436523438, -1.6182715892791748, 0.4269525110721588, 0.7672380208969116, 0.5139446258544922, -0.7642852663993835, 0.5704686045646667, -2.072334051132202, -0.49336859583854675, 0.11345957219600677, 1.0484645366668701, 0.9802424907684326, 1.079638123512268, -0.3622499406337738, -0.6937612891197205, -0.1755535751581192, 0.43112680315971375, 2.070221424102783, -1.0691436529159546, 0.648720920085907, -0.046417221426963806, 0.22251589596271515, -0.3378904163837433, -0.32308635115623474, 0.10927660763263702, 0.4139671325683594, -0.7734287977218628, -1.8863763809204102, 0.02324868179857731, 0.5954208970069885, -0.4901484251022339, -2.2562971115112305, -2.266563892364502, 0.47612181305885315, 0.18790772557258606, -0.007560760248452425, -0.7221455574035645, 0.5408248901367188, -0.010331151075661182, 1.320233702659607, -1.0795506238937378, 1.2587831020355225, 1.3315695524215698, 0.26565247774124146, -0.435964971780777, 0.5727816224098206, 0.6129502654075623, -1.1842262744903564, -1.2536003589630127, 2.6236565113067627, 1.0468636751174927, -0.21139496564865112, -0.12892481684684753, -0.8768252730369568, 0.058930911123752594, -1.1231884956359863, -1.348556637763977, 0.15391160547733307, -2.0912718772888184, -0.9792670607566833, -0.04020986706018448, -1.067665457725525, 0.2397845834493637, 1.286539077758789, 0.07780967652797699, 0.3939913511276245, 0.1418265998363495, 0.077105313539505, -0.7971459031105042, -0.6627119183540344, -0.12176098674535751, -0.33992984890937805, 0.9885132312774658, 0.517596423625946, -0.6542061567306519, -0.5253685712814331, -0.4133802652359009, 0.38961365818977356, 0.001152841723524034, -0.7407393455505371, 0.9519354104995728, -0.7939069271087646, 1.8555656671524048, 0.3523547351360321, 0.9000807404518127, 0.386249303817749, 1.267653226852417, 1.0113266706466675, -0.27411991357803345, 0.6930524706840515, -0.5014510750770569, 0.3916241228580475, 1.5870026350021362, -0.44250455498695374, 0.391735315322876, 0.22912931442260742, 0.11667598783969879, -0.11069577932357788, -1.446917176246643, -0.8393306732177734, 2.2987072467803955, -0.843437671661377, -0.812301754951477, 0.3421359360218048, -0.4279865324497223, 0.6335196495056152, 0.36500635743141174, 0.40494003891944885, -0.3175486922264099, -1.150636076927185, -1.938451886177063, 0.6264628171920776, 1.8171167373657227, -0.29485195875167847, 1.204623818397522, -0.24990029633045197, -0.5132026076316833, 0.9397397041320801, -0.5190287828445435, -0.5433672070503235, 0.16350460052490234, -1.3162214756011963, 0.13842763006687164, 0.9985970258712769, 0.788216769695282, -0.7948733568191528, 0.3547355532646179, -0.6002185344696045, -0.44716575741767883, 0.17729288339614868, -2.3851757049560547, 1.1390607357025146, -0.0020744067151099443, 0.7342290282249451, 0.18857502937316895, -2.0535831451416016, -0.7543984055519104, 0.26902472972869873, 1.3822675943374634, 2.1783103942871094, 0.7610716819763184, -0.6860564947128296, 0.484953373670578, -0.07837475836277008, -0.3528870940208435, -1.7116003036499023, -1.876184105873108, -0.01465670671314001, -0.7662127017974854, 0.629934549331665, -0.3764960765838623, -0.016866378486156464, -0.9481991529464722, 0.4721103608608246, 0.7587487697601318, -0.2675940692424774, 0.5525479912757874, 1.2812471389770508, 0.4024743437767029, 0.23060345649719238, -0.4074086844921112, -0.8415455222129822, 0.627531886100769, 1.010528802871704, -1.483922004699707, 0.3856896460056305, -1.9397987127304077, -0.3014877736568451, 0.11016708612442017, 2.5171706676483154, -0.9946481585502625, 2.511519432067871, 1.041925072669983, 1.2131373882293701, -1.1222491264343262, 0.5082563161849976, -1.1616333723068237, -1.9609342813491821, -0.029835298657417297, -0.20003819465637207, 0.2217741161584854, 1.2925926446914673, -1.1639972925186157, -1.5362775325775146, -1.5038959980010986, 1.4268150329589844, -0.21333865821361542, -0.1436261683702469, -0.4288250207901001, -0.3544084131717682, 0.3697930872440338, 0.4776431918144226, 0.11945830285549164, -1.775795817375183, -0.7262810468673706, -0.7584150433540344, -0.28774985671043396, 1.1055721044540405, 0.692064106464386, -0.4545322060585022, -0.2563590705394745, -1.6455961465835571, -1.3049064874649048, 1.3776386976242065, 0.8522669672966003, -0.23677855730056763, -0.576156497001648, -0.07005466520786285, -0.7431904077529907, -1.1540031433105469, 0.1252385675907135, 0.39832720160484314, -0.3769570291042328, 1.439832091331482, -1.091983437538147, 1.0204768180847168, -1.0090827941894531, 1.572601079940796, -1.4723998308181763, -2.7925665378570557, -0.6807505488395691, -0.384430468082428, 1.437174916267395, 1.0880872011184692, -0.14760294556617737, -0.3935681879520416, -0.9723068475723267, -0.17433398962020874, 0.8296834230422974, 1.7743433713912964, 0.43543919920921326, -1.5632349252700806, -0.25939637422561646, 2.337982416152954, 1.824950098991394, 0.7402545809745789, 0.0828711986541748, 1.6260535717010498, 0.7874492406845093, 0.9889923334121704, 2.1283376216888428, 0.6068584322929382, -0.5410440564155579, 1.3016897439956665, 0.12264492362737656, -0.20942018926143646, 0.2744395434856415, -1.1011857986450195, -1.837182641029358, 0.469072550535202, 0.47311079502105713, -0.8976234197616577, -0.3596894443035126, 0.7933656573295593, -0.18979045748710632, -0.48866006731987, -0.4513469338417053, -1.7232894897460938, -0.07289639115333557, -1.4112324714660645, -1.238910436630249, -0.17128561437129974, -0.5256179571151733, 0.9249986410140991, -0.7580119967460632, -1.4011263847351074, 0.1793886125087738, -0.3730529546737671, -0.5415898561477661, 0.2577141225337982, 1.6714640855789185, 1.4446656703948975, 1.907560110092163, 1.0800151824951172, -0.7608991265296936, 0.7005243897438049, 0.21701982617378235, -0.39719682931900024, -0.8841542601585388, -0.5408271551132202, 0.19691592454910278, -0.5345175266265869, -0.37685444951057434, 0.8443898558616638, 1.3155142068862915, 0.3253975212574005, -0.4354467988014221, 1.1757497787475586, -1.0375622510910034, -0.996540904045105, -1.8454781770706177, -0.8750537037849426, 0.4110090136528015, -1.091481328010559, -1.7165969610214233, 0.44829288125038147, -1.3349789381027222, -3.2674312591552734, 0.6882860660552979, 1.090484619140625, -0.32383736968040466, -1.028928279876709, -0.3828248381614685, -1.463560700416565, 0.7536970376968384, 0.04055599495768547, -1.661399245262146, 0.03227793797850609, -1.0193760395050049, 0.040446892380714417, -0.570305347442627, -1.104773759841919, -0.36267852783203125, 1.3358044624328613, -0.8979368209838867, -1.9792420864105225, 0.6644238233566284, 0.46891847252845764, 0.0277134720236063, 0.5113871097564697, -1.3475022315979004, -0.44727763533592224, 0.7591261267662048, -1.1767395734786987, 0.40284743905067444, 0.23293080925941467, -0.2539113163948059, 0.02129998430609703, 1.2842772006988525, 0.24942190945148468, 0.8738423585891724, 0.9256271123886108, -1.2148106098175049, 0.09033796936273575, -1.6538617610931396, 1.2739671468734741, 0.9946122169494629, 0.8087806105613708, 1.0192869901657104, 0.8509238362312317, -0.454465389251709, -0.47333812713623047, 0.880603015422821, 0.5249532461166382, 0.331440269947052, 0.5380389094352722, 1.821968674659729, 0.7881903052330017, -1.0149871110916138, 0.46508315205574036, 1.0768481492996216, -0.8900042772293091, -2.1510121822357178, 2.790156364440918, 1.9878907203674316, -3.021177291870117, -1.1754578351974487, -1.4129630327224731, -1.335498332977295, -0.4319894015789032, 0.9611131548881531, 0.07985368371009827, -0.6557276248931885, -1.0626354217529297, 0.02004326693713665, 1.740412712097168, 1.8730008602142334, 0.20722486078739166, -0.8359470367431641, 0.23224805295467377, 0.5251664519309998, 0.09025928378105164, 2.3097424507141113, -0.13671259582042694, -0.6589441895484924, -0.784996747970581, -0.8949071764945984, 1.792405366897583, 0.30044984817504883, 0.7786369919776917, -1.0666120052337646, -0.2485329806804657, -1.1248512268066406, -0.8093729019165039, -0.38274431228637695, 0.5361990332603455, 1.032010555267334, -0.28031125664711, 1.1676756143569946, -0.1606086641550064, 0.030629891902208328, -0.11062799394130707, 1.1258816719055176, -1.009026050567627, 0.3998384177684784, -0.1582650989294052, -1.2451112270355225, -0.006261318456381559, -0.19074124097824097, -0.601773202419281, 1.114759087562561, -0.45263969898223877, -0.8473204970359802, 1.3200470209121704, -0.18088608980178833, -0.7758683562278748, 1.2445673942565918, -0.613153338432312, 0.9896857142448425, 0.8521550893783569, 0.8984819650650024, -0.300240159034729, 1.2004337310791016, 0.6618061661720276, 1.2235873937606812, 0.03249626234173775, 0.4583127200603485, 0.9171299338340759, -1.0417343378067017, 0.11999528110027313, -1.1519933938980103, -1.1256707906723022, -0.17193900048732758, -0.363454133272171, -0.031001970171928406, -0.4858301877975464, -0.7283123731613159, -0.4323139786720276, -0.013784999959170818, -0.18064658343791962, -1.0236245393753052, 0.15008249878883362, 0.17378173768520355, -0.4638289511203766, 0.9125618934631348, -0.23416674137115479, -0.21891163289546967, -0.11597301065921783, 2.385873317718506, 1.1168442964553833, -0.14225099980831146, 2.253735303878784, 0.9389410018920898, -0.1634591668844223, -1.5980843305587769, -0.5909850001335144, 0.9222814440727234, -0.618723452091217, 1.2476118803024292, 0.1151951476931572, -0.6685547232627869, -1.1461677551269531, 0.10900875926017761, -0.7908079028129578, -0.18878090381622314, -0.1920221894979477, -0.5611708164215088, -0.4649636149406433, -0.3176233172416687, 1.0152852535247803, -0.32725414633750916, 1.4340486526489258, 0.2436143159866333, 0.9417625665664673, -0.14243891835212708, -0.5425183176994324, -0.18690361082553864, -0.820232093334198, 1.8991464376449585, 1.5109654664993286, 1.5466231107711792, -0.4098077416419983, -0.8716021180152893, 0.15186065435409546, -0.4699948728084564, 0.8002009391784668, -0.5968831777572632, -0.061617087572813034, -0.23445361852645874, 0.9506785273551941, 1.018571376800537, 0.781201183795929, -1.2195565700531006, -0.45707017183303833, -0.3174538314342499, 2.0731494426727295, 0.09931321442127228, -0.4056679308414459, 0.9824268221855164, -0.7874475717544556, 0.23561187088489532, -0.32809144258499146, 0.40918391942977905, -1.2465578317642212, 0.33212724328041077, -0.4826626479625702, -0.3206499516963959, -0.6003472805023193, 0.24203656613826752, -0.2912772595882416, -0.13271024823188782, -0.9099090695381165, 1.6718913316726685, -0.7079047560691833, 1.5958061218261719, 0.7524877190589905, 0.6100393533706665, 2.095020055770874, -0.7631176114082336, -0.42262160778045654, -1.4632205963134766, -0.05625385046005249, -0.8237460851669312, -0.5639963746070862, 1.631902813911438, -0.49700891971588135, -0.3879166543483734, 0.5154153108596802, -0.37290096282958984, -0.5525978803634644, 0.5612038969993591, -1.202552318572998, -1.5048999786376953, -0.5156913995742798, 0.6387205123901367, -1.407561182975769, -0.6561412811279297, 0.22969838976860046, 0.6678948402404785, -0.7488875389099121, 0.5212708115577698, -0.661757230758667, -0.9745112061500549, -1.5286563634872437, -0.6009125709533691, -1.0614094734191895, 0.20470936596393585, 1.5358930826187134, 1.7225360870361328, -1.576072096824646, 0.5286346077919006, -0.09215329587459564, -0.16508115828037262, 0.714250922203064, -1.251724362373352, -0.06465298682451248, -0.4764242470264435, 2.018778085708618, 0.2318495661020279, -1.6577736139297485, 0.7102546095848083, 0.5660727024078369, 1.6163207292556763, -0.7102009057998657, 0.2802075147628784, 0.5574367046356201, -0.7864266037940979, 0.7278173565864563, -0.18832814693450928, -0.37463921308517456, 0.08348910510540009, -1.3054558038711548, -1.1101617813110352, 1.0755287408828735, 1.8045603036880493, -1.3164443969726562, 0.9402625560760498, -1.2583820819854736, 0.06522087752819061, 0.35213184356689453, -0.3391597867012024, -0.9158684015274048, -0.2932184040546417, -0.24376294016838074, -0.4629661440849304, 0.6116859316825867, 1.2492091655731201, 0.7827233076095581, -0.5499243140220642, -0.19373564422130585, -0.4198874235153198, -0.89956134557724, 1.6240965127944946, -0.025116151198744774, 1.4542996883392334, 2.1340205669403076, 0.6541118621826172, -0.17166094481945038, 0.5665009617805481, -0.4488973915576935, 1.949928641319275, 0.04029485583305359, 0.8286519646644592, 0.7695297002792358, -1.4518964290618896, 0.3206078112125397, 0.8008856177330017, 0.9593398571014404, 0.36534324288368225, -1.164936900138855, -0.3291865885257721, -3.197423219680786, -0.2320658266544342, 2.2650856971740723, 0.9422969818115234, -0.49375659227371216, -0.6869304180145264, -0.8773172497749329, 0.2169688642024994, 1.0270837545394897, -0.3808877170085907, -0.8029876351356506, -1.3987107276916504, 0.9560277462005615, -0.49281632900238037, 1.068955421447754, -0.4223101735115051, -0.47083526849746704, -0.7526910901069641, -1.8331537246704102, 0.40309491753578186, -0.6088058352470398, -0.18223664164543152, -0.7600995898246765, 0.5587124824523926, 0.3032960891723633, -1.0383342504501343, 0.8840446472167969, -0.036010220646858215, 0.060486823320388794, -0.3945499062538147, -0.853824257850647, 0.586032509803772, 1.3381537199020386, 1.2949104309082031, -1.2927123308181763, -1.2223209142684937, -1.9115371704101562, 1.6846803426742554, -1.3102892637252808, -2.0776479244232178, -1.065614938735962, -2.0361664295196533, 0.6864210367202759, -1.206438660621643, -0.537556529045105, 2.047919750213623, 0.2579856216907501, -3.085240125656128, -1.0736562013626099, -0.8900302052497864, 0.5082488656044006, 0.8965453505516052, -0.6760367751121521, -0.22955012321472168, 0.7796162962913513, -0.5143357515335083, 0.0853152871131897, -1.0777053833007812, -0.6010659337043762, 2.107304334640503, -0.842481255531311, -0.37157386541366577, 1.1820036172866821, 0.5380580425262451, 2.512566089630127, -0.44575396180152893, -0.07685523480176926, -1.7542997598648071, -0.49269381165504456, 2.425630807876587, 1.56044340133667, 2.4655511379241943, 0.36211228370666504, -0.12316560745239258, -0.36903250217437744, -0.6259045004844666, -1.5859583616256714, -0.6858001351356506, -0.12654174864292145, 0.8872596621513367, 1.9922236204147339, -1.2082403898239136, 0.7830051779747009, -0.3292309641838074, -1.4272003173828125, 0.5945072174072266, 0.6398154497146606, 0.8709239959716797, 0.6931217312812805, 0.5811781287193298, -0.12116538733243942, -0.33998769521713257, 1.1449294090270996, 0.5518524050712585, 0.7108882069587708, 0.11297550052404404, 1.509390115737915, -0.09137379378080368, 0.16065320372581482, -0.4977042078971863, -0.6367749571800232, 0.6124299764633179, -0.69181889295578, -1.4685150384902954, 0.4762907028198242, -0.008643433451652527, -0.47354230284690857, 0.30289945006370544, 0.3072645366191864, 0.31184741854667664, -0.9001712203025818, -0.5702089667320251, -0.24940162897109985, 0.338527649641037, -1.4853304624557495, 0.9272831678390503, 0.6175644993782043, 0.3579038381576538, -0.4515831172466278, 0.7205889225006104, -0.0064501212909817696, -0.3577738404273987, 0.37365075945854187, 0.43679699301719666, -1.2984638214111328, 0.2008354365825653, -0.700792133808136, 0.4211263954639435, -1.104469656944275, -1.2295193672180176, -1.3334729671478271, 1.5318955183029175, -1.7714449167251587, -0.6169958710670471, -1.6936862468719482, 0.4895785450935364, -1.2013856172561646, -1.1877278089523315, 0.7728338241577148, -0.5493716597557068, -0.5474534034729004, 0.7207757234573364, 0.9862257242202759, -1.0448704957962036, 0.37389180064201355, -0.19962982833385468, -0.6765841841697693, 0.8643040657043457, -1.4737040996551514, 0.180297389626503, 0.9144929051399231, -1.661658525466919, 0.993249237537384, 0.9144186973571777, 0.409511536359787, -1.9776448011398315, 0.9656352400779724, 1.0590490102767944, -1.1480506658554077, -0.25536420941352844, -0.4991474151611328, 1.4384779930114746, -2.5492496490478516, 1.4681322574615479, 1.102806806564331, -0.2469189614057541, -0.84064120054245, 0.2189914584159851, -0.12116004526615143, -0.3734744191169739, 1.2951364517211914, 0.8915730714797974, 1.1515253782272339, 0.8231952786445618, 0.5840669870376587, -0.6061139702796936, 0.44258350133895874, 0.01489738654345274, 2.2701878547668457, -0.32659634947776794, -0.2133370339870453, 0.6288662552833557, -0.2874818444252014, 1.0134341716766357, 1.1194497346878052, 0.12739261984825134, 0.019131792709231377, -0.22954915463924408, -0.40084323287010193, 0.6225606203079224, 0.4002251923084259, -0.6863791346549988, 1.1992899179458618, -0.6990585327148438, -0.4190099537372589, -0.8630900382995605, -0.4719081223011017, 1.2195816040039062, 0.7825882434844971, -0.0962417721748352, -1.9127752780914307, -0.5481735467910767, 0.3401852250099182, 0.4785555601119995, -2.1978275775909424, 0.15032248198986053, -1.1915655136108398, 1.0704525709152222, 0.20936380326747894, -1.5925403833389282, -0.08311943709850311, 0.9971930384635925, -2.220226526260376, -0.5127075910568237, -0.013376223854720592, -0.044285740703344345, 0.6052053570747375, 0.9316248297691345, 1.0057882070541382, 2.9158778190612793, 0.3404509723186493, -0.6450645327568054, 0.13306273519992828, 0.9504312872886658, -0.3324379324913025, -0.15832079946994781, -1.7763636112213135, -0.20021581649780273, 1.2009303569793701, 0.8686173558235168, 1.0562548637390137, 1.8271287679672241, 0.2409796267747879, -1.3685497045516968, -0.589876115322113, -0.8322100639343262, 0.5412876605987549, -0.5584167838096619, -0.46176210045814514, -0.5199220180511475, 0.07705796509981155, -0.7374114394187927, -0.1845770627260208, 0.6301808953285217, 0.10941923409700394, -1.6164836883544922, -0.05487874522805214, 1.2491459846496582, -0.30492469668388367, -0.2058643251657486, -1.8800870180130005, 1.1515597105026245, -0.7221347093582153, 1.0790268182754517, -0.18383102118968964, -0.5766919255256653, -0.8061294555664062, -0.20045050978660583, 0.5348878502845764, -1.4817719459533691, -0.32050880789756775, 0.8347111344337463, -0.37367066740989685, -0.19534577429294586, -0.30092552304267883, -0.25133436918258667, -1.1324318647384644, -0.14778128266334534, 0.9987969994544983, -1.4194755554199219, 2.381544828414917, 0.8379960656166077, -0.11814092844724655, 0.7770281434059143, 0.5474526882171631, -0.010607785545289516, 0.3962072730064392, 0.5601469278335571, 1.8499586582183838, 1.0601052045822144, 0.8245868682861328, 0.18441849946975708, -0.3131469488143921, -0.10798955708742142, 0.8544285893440247, -1.5976868867874146, 1.0454474687576294, 0.4017762839794159, -0.21143972873687744, 1.0165694952011108, -0.13172554969787598, -0.5323254466056824, 0.30853453278541565, -0.4623299539089203, 1.5026715993881226, -1.2204989194869995, -0.33802416920661926, -0.4709620177745819, -0.7168704271316528, -0.4045206308364868, -2.160841703414917, -0.4874936640262604, 2.3718173503875732, -0.8177666664123535, 1.239770531654358, -1.2247693538665771, -1.6455484628677368, 0.8016343712806702, -1.9247629642486572, -0.26272594928741455, 0.2770603895187378, -0.9848967790603638, 1.056522250175476, 0.6868582367897034, 0.3008703887462616, 1.3024752140045166, 1.0628514289855957, 0.6736735105514526, -0.7600495219230652, -0.06567877531051636, -0.6397550106048584, -2.4223954677581787, 2.236696720123291, -1.3779653310775757, 1.5709037780761719, 0.8744381666183472, 0.8817350268363953, -0.1786702573299408, -0.66612309217453, 0.10916406661272049, -1.0534688234329224, -0.04489050433039665, 1.521683692932129, 0.395288348197937, -1.240609049797058, 0.22065232694149017, -1.3033411502838135, 0.32726937532424927, -0.45626965165138245, 0.03696993738412857, -1.8439778089523315, -0.8637722134590149, 0.4615117311477661, -1.0852041244506836, -1.3907183408737183, -0.8531891703605652, -0.8788555860519409, -0.20846857130527496, -0.23294945061206818, 0.4213595986366272, -0.6575110554695129, -1.0438590049743652, 0.7124961614608765, 0.4457707405090332, 0.2843839228153229, 1.8615365028381348, 0.166897252202034, -0.3861250877380371, -0.6397836804389954, 0.34746813774108887, 0.4461624324321747, -0.7026980519294739, -0.338613361120224, 0.15293839573860168, 1.2664713859558105, 0.059297893196344376, -1.3859670162200928, -0.549704909324646, -1.4708465337753296, -0.19093528389930725, -0.4036587178707123, -0.13583621382713318, 0.7086982727050781, -0.4292715787887573, 0.11624275892972946, -0.6893202066421509, 0.49024122953414917, 1.9290521144866943, 1.8058537244796753, -1.1618942022323608, 1.1962611675262451, 0.5822452306747437, 2.1645684242248535, 0.40454867482185364, -1.0515632629394531, -0.7261607646942139, -0.6620243787765503, -0.26273131370544434, -1.0753822326660156, -0.4252443015575409, -0.11705829203128815, -0.7733404040336609, -1.4481111764907837, -0.682613730430603, -0.9207446575164795, 1.3405873775482178, 0.5229364037513733, -1.151065707206726, -0.00819437112659216, 0.3936767876148224, 0.27680155634880066, 0.27625468373298645, 0.710588812828064, 0.17151691019535065, -1.1988834142684937, -0.7912791967391968, -0.40921148657798767, 0.6520221829414368, -0.22841273248195648, -0.13858599960803986, 0.20815445482730865, 0.20288844406604767, 0.3993369936943054, -0.5482693314552307, 0.03493509814143181, 0.25769487023353577, -0.1303066611289978, 0.07976092398166656, -0.12416285276412964, 0.6450074315071106, 0.38866737484931946, -1.0295507907867432, -0.16031880676746368, -0.24224169552326202, 0.43160220980644226, -0.2678838074207306, -0.6213324666023254, 0.3530489504337311, 0.056906379759311676, 0.8436574339866638, -0.6099944710731506, -1.3964416980743408, -2.0264716148376465, 0.20234648883342743, -1.1672334671020508, -0.27507349848747253, 0.38909801840782166, 0.15758928656578064, -0.6381781697273254, -0.9362547993659973, 0.5713061094284058, 0.4680916965007782, 1.055863857269287, -0.13357718288898468, 0.20091751217842102, 0.4901941120624542, 1.0072931051254272, 0.050710540264844894, 1.8307955265045166, 0.1737125962972641, -0.21719828248023987, -0.20839016139507294, -1.3841636180877686, 1.434653878211975, 1.447984218597412, -0.9744012355804443, 1.4536906480789185, -0.2011757493019104, -1.2874783277511597, -1.618841528892517, -0.4132808446884155, -0.4020792841911316, -0.5652344226837158, -0.6868815422058105, -0.5420117974281311, -0.40912142395973206, 0.24243168532848358, -0.9227097034454346, -0.3228141665458679, -1.7046154737472534, 1.101482629776001, -0.2261657565832138, -0.30921489000320435, 0.5886233448982239, -0.7989237308502197, 0.37482884526252747, -0.07630787789821625, 1.8377752304077148, 0.2415417581796646, -0.3550766706466675, -0.24435453116893768, -1.7286454439163208, 2.9015164375305176, 0.12486990541219711, -0.1771988570690155, -0.8942481279373169, 0.4052905738353729, 0.2295648157596588, -0.5132634043693542, 0.13812346756458282, -0.3527545630931854, -0.09048672765493393, 0.31094175577163696, 0.08824162930250168, 0.8246771097183228, -0.37742748856544495, 0.7928188443183899, -0.20965956151485443, -1.1233104467391968, 1.1359361410140991, -0.40621399879455566, 0.12998992204666138, 0.2483295351266861, 0.7662431597709656, -0.055982187390327454, 0.7450763583183289, -1.6917476654052734, 1.0220654010772705, -0.44159048795700073, 0.01835772581398487, 0.8394890427589417, -1.85585618019104, -1.1743066310882568, 1.2134770154953003, 0.39141613245010376, 0.8843033313751221, 0.9228286743164062, -0.6844433546066284, 0.8382556438446045, -1.093171238899231, 0.6435971260070801, -0.24331237375736237, -0.12818388640880585, -0.2937633991241455, 0.48312073945999146, 0.041918594390153885, 1.121375560760498, 1.3641003370285034, 0.2500375509262085, -0.6022546887397766, 2.6892130374908447, 2.6091227531433105, 1.4829868078231812, 0.9327779412269592, 1.4604798555374146, 0.080167755484581, 0.70055091381073, -0.8741149306297302, -1.2423523664474487, -0.014113969169557095, 0.7927822470664978, 0.7727216482162476, 0.6628018617630005, -0.0910845622420311, 0.4290519654750824, 0.022354111075401306, 0.2654290795326233, -0.2509424090385437, 0.0029100237879902124, -0.28197288513183594, -0.9891958832740784, 1.1645587682724, 1.509636640548706, 1.8835241794586182, -1.0190235376358032, 0.04362878575921059, 0.021400632336735725, 0.43756037950515747, -0.31350675225257874, -1.3860197067260742, 0.12161029130220413, -0.5202368497848511, -1.1720737218856812, -0.5388950109481812, -0.16414126753807068, 0.2512098550796509, -0.6618878245353699, 0.784990668296814, -1.708183765411377, -0.2774176001548767, -1.1554840803146362, -0.12092790007591248, 0.06616753339767456, 1.4153056144714355, -0.19742678105831146, 0.05933988466858864, -1.7762248516082764, 1.3265036344528198, 1.1177424192428589, -0.15043531358242035, 1.833373785018921, -1.0938429832458496, -0.4373902976512909, -1.4389872550964355, 0.0729164332151413, 1.156264066696167, 1.0021052360534668, 1.1303786039352417, 0.9226210117340088, -0.6012166738510132, -0.39058488607406616, -0.7522111535072327, -0.646210253238678, -1.201682448387146, 1.0487529039382935, -0.4836808443069458, 0.7112727165222168, 1.4610992670059204, -0.2774108648300171, 0.848540723323822, 0.9131507277488708, -0.5594849586486816, -0.24787382781505585, -1.1757464408874512, 1.7817420959472656, 1.7238078117370605, 0.07451160252094269, -0.7586610913276672, -0.551202118396759, -1.5833702087402344, -0.021942278370261192, -1.1696099042892456, -1.0165174007415771, -0.06791581958532333, 1.2314624786376953, 0.35487663745880127, -0.7565487027168274, 1.9020406007766724, 0.4390394389629364, 0.9696183800697327, 1.0615540742874146, -1.0513784885406494, -0.22992604970932007, 1.793521761894226, 0.2820863425731659, 0.8042874932289124, -0.7187666893005371, -0.11181633174419403, -0.5473441481590271, 0.33826136589050293, 0.3142956793308258, -0.36783847212791443, 0.1334165334701538, 0.8284748792648315, -0.19607770442962646, 0.43490487337112427, 0.9103896021842957, -0.30607083439826965, -1.1217907667160034, 0.9637740254402161, 2.19907283782959, 0.43545177578926086, -0.359596848487854, 0.2739261984825134, -2.015230178833008, -0.34184983372688293, 0.3214990794658661, 0.8660144209861755, -1.5837886333465576, -1.4641634225845337, 1.310389518737793, 1.0798194408416748, 0.4811806082725525, 0.6898097991943359, 1.4353642463684082, -0.3431673049926758, 0.6341519355773926, 0.8259078860282898, 1.0169565677642822, 0.4126073718070984, 1.4598848819732666, 0.952003002166748, -0.7945813536643982, -0.8803603053092957, -0.2654556930065155, 0.19933047890663147, 0.9464505314826965, 0.32716986536979675, -2.551259994506836, 1.9300886392593384, -0.711037278175354, 0.004205118399113417, -2.406041145324707, 1.0513454675674438, 1.3270642757415771, 0.5409901142120361, -0.39601466059684753, 0.11384371668100357, 1.8999797105789185, -0.53206467628479, -0.859420895576477, 0.06884469091892242, -0.4786319434642792, 0.4991527199745178, -2.262463331222534, -0.6644049882888794, 1.559767484664917, 0.7067237496376038, 1.3901079893112183, -0.3242948055267334, 0.7471365928649902, -2.60986065864563, 0.4845174551010132, -0.5646410584449768, -0.0026550914626568556, 0.9689953923225403, -0.4461330473423004, 0.18935759365558624, 0.2616463899612427, 0.15102995932102203, 0.13876710832118988, 1.8134377002716064, -1.2569611072540283, 1.0637543201446533, 0.08037129044532776, 0.15594729781150818, -0.2827288508415222, 0.5962924957275391, -0.906923770904541, 0.48354023694992065, 0.8513494729995728, -1.3357161283493042, -0.30337706208229065, 1.0310806035995483, 0.7504709362983704, 0.0842253714799881, 0.5877379179000854, -0.26478925347328186, -0.5551730990409851, -1.8455311059951782, 0.2818453311920166, 0.03653861954808235, 0.7088054418563843, 0.14991404116153717, 0.08979208767414093, 1.0725144147872925, -1.2413848638534546, 0.5160589218139648, -1.0069471597671509, 0.580400288105011, -0.7920028567314148, 0.667216420173645, 0.16714483499526978, 0.30088162422180176, 0.4403863251209259, -0.6741024851799011, -0.3153105080127716, -0.16042926907539368, 1.1479324102401733, 0.3429337441921234, -1.2778899669647217, 1.427141785621643, -0.18409433960914612, 1.2655627727508545, 0.059947118163108826, 0.36551356315612793, 1.1264642477035522, 2.116678237915039, -0.6606360673904419, 0.9992788434028625, 2.1858367919921875, -1.1151363849639893, -0.6954604387283325, -1.0686614513397217, -0.5530418753623962, 0.7463154196739197, -0.5735496878623962, -1.9449447393417358, 0.025984464213252068, 1.22359299659729, -1.5987122058868408, 0.8553666472434998, 0.736350417137146, -0.5237066149711609, -0.29347437620162964, 2.014296293258667, 0.36736971139907837, -1.3559000492095947, -0.1976555436849594, -0.688721239566803, -0.5982056856155396, -0.8018976449966431, -0.4883279800415039, -0.46533313393592834, 0.07722480595111847, -0.8033127784729004, -1.6345713138580322, -0.6378790140151978, -0.2303425520658493, 0.045746397227048874, -0.18072821199893951, 0.43059197068214417, 0.07791728526353836, -1.2595123052597046, -1.0215474367141724, -0.42516613006591797, 1.6212663650512695, -0.3313254415988922, -0.885280191898346, 0.6044290065765381, -0.1483500897884369, 0.3716890811920166, 0.49222299456596375, -0.15744651854038239, 0.17204086482524872, -1.4243626594543457, -0.39714956283569336, 1.0849179029464722, -1.5946929454803467, -0.46373793482780457, 0.7488438487052917, 0.4692612290382385, -1.3654276132583618, -1.7897394895553589, 0.4975270926952362, 0.4943447411060333, -0.31263431906700134, 1.32307767868042, -0.2903774082660675, -0.9836138486862183, 1.2344028949737549, 0.9933933019638062, 0.24466295540332794, 0.040866341441869736, 0.2637424170970917, -2.139474630355835, 0.6632161736488342, 0.3179500699043274, -0.7638258934020996, 1.0041749477386475, -0.012689848430454731, 0.9537270665168762, 0.8980122208595276, -0.2480442374944687, -0.5988965034484863, 0.2795964479446411, 1.213640809059143, -1.5441030263900757, 0.06471279263496399, 0.2746315002441406, -1.870772123336792, 1.8611820936203003, -0.9683079719543457, 1.0008959770202637, 0.23488080501556396, 0.28462859988212585, 1.3047711849212646, -0.26380228996276855, 2.2661056518554688, 0.4660395681858063, -1.720889687538147, -1.3438255786895752, -0.9296898245811462, 0.7677770256996155, -0.1649046093225479, -0.1352159082889557, 2.971187114715576, -1.7828892469406128, -2.3496174812316895, -0.8969369530677795, 2.709075450897217, 0.19601362943649292, -1.0466668605804443, 0.527831494808197, 1.0819021463394165, -0.20282503962516785, -0.47237902879714966, -0.46040070056915283, 0.9777954816818237, -0.11931386590003967, 0.8773460984230042, 0.07166696339845657, -1.7586075067520142, -0.8053646087646484, -0.854403018951416, -1.2114617824554443, 0.683678388595581, 0.6211780309677124, 0.14323748648166656, -0.7180012464523315, 0.13867205381393433, -0.11053327471017838, 0.565963625907898, -0.09045807272195816, 0.4895855188369751, 0.3147056996822357, 1.0951262712478638, -0.9123926162719727, 0.3209744095802307, -0.8921770453453064, -0.927342414855957, 0.6213961243629456, -0.0178761538118124, 0.6176143884658813, 0.9235701560974121, -2.167476177215576, 0.9136724472045898, 0.7672975063323975, -1.1054290533065796, -0.2099536806344986, 0.48333704471588135, 0.8972133994102478, 0.06601466238498688, 0.5289061665534973, -0.1706792265176773, 0.3350861966609955, -0.3142493665218353, -0.3984310030937195, -1.3959784507751465, 0.9880074858665466, 1.9754211902618408, 0.8633911609649658, -1.6747684478759766, 0.4155435264110565, -1.0458109378814697, -1.208590030670166, -0.9414167404174805, 0.3470759689807892, 2.128760576248169, -0.6195902824401855, -1.4569402933120728, 0.32248446345329285, 0.13357791304588318, 0.23149988055229187, -0.23326139152050018, 0.9571572542190552, -0.14422011375427246, -0.283833771944046, -1.9789611101150513, -0.3094383776187897, -0.27300965785980225, 0.0812024399638176, 0.8823996186256409, 0.13582965731620789, -2.242258310317993, -2.059937000274658, 0.673759937286377, -0.9077293276786804, -0.47405895590782166, -0.12480760365724564, 0.089272640645504, -0.5660448670387268, -0.17711296677589417, 0.1993512511253357, 0.8668360710144043, 0.5274118781089783, -0.14799179136753082, -0.2375420331954956, 0.17619691789150238, -1.019173502922058, -0.33675140142440796, -0.3393041789531708, -0.6058424115180969, -0.08660893887281418, -0.6346612572669983, 0.9599772691726685, 0.1122613400220871, 0.4709872603416443, 2.1070029735565186, 0.010153498500585556, -0.834854245185852, 1.0983160734176636, -0.6956260800361633, 0.4182102084159851, -0.5272231698036194, 1.558438777923584, -0.7731626033782959, -0.258466899394989, 0.8300668597221375, -0.3528618812561035, -0.9444526433944702, -0.4804325997829437, 0.3828882873058319, 0.7159587740898132, 0.13755130767822266, 0.23564371466636658, 0.6358039379119873, -0.6666387915611267, 0.9431325197219849, -0.5962768197059631, -1.6588222980499268, -0.28887853026390076, -0.638837993144989, 0.3645007610321045, 0.458907812833786, 1.8355716466903687, 0.6556342840194702, 1.6343954801559448, 2.620748519897461, 0.06035292148590088, 0.949069082736969, -1.1164634227752686, 2.0128836631774902, -1.7501226663589478, -0.5739983320236206, -1.312820315361023, -1.686279058456421, 0.2344556301832199, -0.8671186566352844, 1.797980546951294, 1.0184251070022583, -1.6559489965438843, -0.5777209401130676, 0.6635436415672302, -0.2875919044017792, -0.3013427257537842, 0.7132595181465149, -0.8809494972229004, -1.988886833190918, -0.7868351340293884, -2.010262966156006, -1.1401218175888062, -0.09778626263141632, 0.25687986612319946, -0.9829402565956116, 1.3534259796142578, 0.529716968536377, -0.7794011831283569, -1.3997657299041748, 0.38283464312553406, 1.1461764574050903, 0.2579432427883148, 0.11596544086933136, 1.0263005495071411, 0.9513402581214905, 1.0460894107818604, -0.7687765955924988, -0.3032398521900177, 0.8357940316200256, 0.7360212206840515, 0.7387682795524597, 0.7156460881233215, -0.7538806200027466, -0.0453900508582592, -0.6642351746559143, -0.6500027179718018, -0.6822222471237183, -1.4112330675125122, -0.21054545044898987, -0.5260010361671448, 1.5670280456542969, 0.4444611072540283, -1.4256482124328613, 0.21387487649917603, 0.3459315299987793, 0.05782483518123627, 1.3788551092147827, -0.31812331080436707, 0.05645507946610451, 1.4823391437530518, -0.537725567817688, -1.1112477779388428, 0.3499803841114044, -0.8999171853065491, -1.579723596572876, 1.2451947927474976, 0.3853326141834259, -1.3190046548843384, -0.824198305606842, -0.5341749787330627, -0.9999505877494812, 0.6254498362541199, -1.2567301988601685, -1.122046709060669, 0.009378906339406967, 0.7878450751304626, 3.2825396060943604, -0.11670531332492828, -2.700612783432007, -0.22386787831783295, 1.3479549884796143, 1.0888311862945557, 0.36873823404312134, 1.5954651832580566, 1.3480569124221802, -0.7580240964889526, -0.215918630361557, -0.08148246258497238, 0.4223296642303467, 0.5808550119400024, 0.25493085384368896, -0.8185977339744568, -0.3198215961456299, -1.483744502067566, 1.1784571409225464, -0.7030459642410278, 0.07712041586637497, 0.27937087416648865, 0.8361589312553406, 1.3332502841949463, 0.3853529095649719, -0.7749696373939514, -0.43589431047439575, 0.6184977889060974, -0.36187484860420227, 0.8521261811256409, 0.6072938442230225, 0.6987863183021545, 0.4073452055454254, -0.1480453759431839, 0.9213875532150269, 1.9304612874984741, -0.22329290211200714, 0.127630352973938, 0.860592782497406, 0.18784315884113312, -0.6713162660598755, -0.927738606929779, 1.1337648630142212, 0.8911373615264893, 1.558409333229065, 1.5712578296661377, -1.0720242261886597, -0.758192777633667, 1.6310263872146606, -0.8102781772613525, 0.607400119304657, -1.1809064149856567, 1.1351507902145386, 0.7314459681510925, 0.12365119159221649, 0.7954714894294739, -0.20916134119033813, 1.1206077337265015, 0.5886415839195251, -2.169090509414673, -0.08790173381567001, -0.4135516285896301, 0.31395503878593445, -0.24314278364181519, -1.440608024597168, 0.9142798185348511, 0.007852144539356232, -0.2019854038953781, 0.5157341957092285, -0.05683545023202896, 0.7772846221923828, -0.32274413108825684, 0.3371034860610962, 0.15971636772155762, 1.1906465291976929, 0.919121265411377, -0.47723063826560974, -0.014357890002429485, 1.4989070892333984, 0.8029950261116028, 1.8962912559509277, 0.7196400761604309, 0.9825453758239746, -0.30036604404449463, 0.7548298239707947, -2.2415964603424072, -0.3752228021621704, -0.2116311937570572, 1.4118444919586182, 0.7543107867240906, -0.45630091428756714, -0.5816650986671448, -0.20380376279354095, 0.7692894339561462, 0.7511840462684631, 0.40150612592697144, -0.5278804898262024, 0.6855437159538269, -0.921229898929596, -0.7095507979393005, -1.1707559823989868, -0.17930907011032104, 0.6989723443984985, 0.026888394728302956, 1.4941966533660889, 0.08154843002557755, 0.2302708625793457, -0.013341418467462063, -0.6097922921180725, -1.524206519126892, 1.231067419052124, -0.7364915013313293, 1.4452482461929321, -0.8324069380760193, -0.8258392214775085, 2.5400278568267822, 1.5266392230987549, 1.5006953477859497, -0.6437927484512329, -1.727483868598938, 0.24706661701202393, 0.04384884983301163, -1.2565762996673584, -0.4660770297050476, 0.28690841794013977, -0.5107067227363586, 0.015021879225969315, -0.7636079788208008, 0.44835174083709717, -0.8973925113677979, -1.6051504611968994, -0.12020626664161682, 0.3035884201526642, 0.6519176363945007, 0.6247398853302002, -1.1338502168655396, -1.2373371124267578, 0.6676783561706543, -0.3221237063407898, -1.0930789709091187, 0.642312228679657, 0.6854863166809082, -1.450379490852356, 1.7042467594146729, 1.2913212776184082, 0.8248606324195862, 0.657241940498352, 0.2430962324142456, -0.228378564119339, -0.8930649161338806, -0.9179550409317017, 0.4232414662837982, 2.2637979984283447, 1.343268871307373, -0.41930529475212097, 0.7177938222885132, 1.8396289348602295, 0.881706714630127, 0.44135135412216187, 1.8260077238082886, -0.08499445021152496, -0.2628181278705597, 0.4045897126197815, 1.8561416864395142, 0.25443851947784424, -0.26316559314727783, 2.354029417037964, -0.24020999670028687, 0.7635588049888611, -1.3650728464126587, -0.4986569285392761, 0.2035616785287857, -0.3938322961330414, -0.16887058317661285, -0.3431532382965088, -2.027935266494751, 0.09434075653553009, -0.9257756471633911, 0.9227330088615417, 1.9049880504608154, 0.08552394062280655, 1.9288389682769775, 0.45817863941192627, -0.6876744031906128, -1.0289537906646729, -0.3785306513309479, -0.5964143872261047, -1.9147844314575195, -1.1693363189697266, -0.18729889392852783, 0.6303409934043884, 0.9863729476928711, -0.3141252398490906, 0.3669412136077881, 0.14151963591575623, -1.195134162902832, 0.2702319622039795, 0.06981302797794342, -0.8021724820137024, -0.13440802693367004, -1.0013707876205444, 0.81312495470047, 1.9816938638687134, -0.21127937734127045, 1.1720610857009888, 0.4208189845085144, -0.16035781800746918, 0.8206865787506104, 0.6114331483840942, 0.7396600842475891, -0.6488701701164246, -1.9741727113723755, -0.6690479516983032, -0.7007823586463928, 0.8268990516662598, 0.7915893793106079, -1.1724485158920288, 0.857388973236084, 0.8302123546600342, -1.1609137058258057, -0.870465874671936, 1.6943873167037964, -0.9399497509002686, 0.15366177260875702, 0.6046481728553772, 1.2751110792160034, -0.03617629408836365, -0.4500961899757385, 0.6104171276092529, 0.6839226484298706, 0.19729113578796387, -1.3243948221206665, -0.39950284361839294, -1.4201594591140747, -0.6973406672477722, -0.2363337278366089, 1.0137526988983154, 0.3079782724380493, 0.4961957335472107, 0.37757259607315063, -0.5871328115463257, 0.8468865156173706, 1.6427900791168213, -0.009652312844991684, -0.9864449501037598, 0.9113908410072327, 1.3174479007720947, -0.12247008830308914, -1.6926788091659546, 0.543878436088562, -1.6578404903411865, -1.084505319595337, -1.3153973817825317, 0.3819590210914612, 0.20659036934375763, -0.4387679696083069, 0.9316399097442627, -0.6845875382423401, -2.0785224437713623, -0.6622335314750671, -0.3232342302799225, -0.879223108291626, -0.64077228307724, -0.09215408563613892, 0.38517317175865173, 0.10252702236175537, -0.5439684391021729, -0.40148258209228516, 0.10491765290498734, 0.4051486551761627, 1.0275626182556152, 0.06685622781515121, 2.0125880241394043, 0.07732567936182022, 0.6528477668762207, -0.7863008975982666, 0.3264354467391968, 0.7746242880821228, 1.4270960092544556, 0.2883283793926239, 0.07638402283191681, -0.44464564323425293, 1.21699857711792, 2.890023708343506, -1.1398718357086182, 0.2656748592853546, 0.587973415851593, 0.5785131454467773, -0.5692815184593201, 1.5158793926239014, 0.3419129550457001, -1.215287208557129, 0.26554253697395325, 2.05946946144104, 0.5168347358703613, -2.254535436630249, 0.7344229221343994, -0.6338294148445129, 1.1643646955490112, 0.003043669741600752, -0.3832522928714752, -1.0881699323654175, 0.7770740389823914, 0.41712531447410583, -1.7179656028747559, 1.0551538467407227, -1.545501708984375, -0.5773816704750061, -0.3426383435726166, 0.3873879313468933, -0.006140603683888912, 0.24249547719955444, -0.20279155671596527, 1.1314319372177124, 0.6729838848114014, 0.4214443564414978, -0.11332479119300842, -2.2333335876464844, -0.6855576038360596, 0.7705621123313904, 1.1572413444519043, 1.466997742652893, 0.16082817316055298, 0.10796379297971725, -1.4900026321411133, -0.6589651107788086, 0.9423607587814331, 0.3836934566497803, -0.08393647521734238, -0.24934686720371246, 0.4441007971763611, 0.5849788188934326, 0.5843559503555298, -0.8610277771949768, -0.9777320623397827, -1.7317872047424316, -1.4387418031692505, 0.7472278475761414, 0.9462347626686096, -0.4640766382217407, 0.6915079951286316, 1.0223900079727173, 1.4794896841049194, 1.5349655151367188, 0.7038491368293762, -0.8070816993713379, 0.08461810648441315, 0.3601989150047302, -0.3902727961540222, -0.8004055619239807, 0.3919145166873932, 0.8911592960357666, 1.2313683032989502, 0.5479575991630554, -1.4460258483886719, 1.8260780572891235, 0.9179889559745789, -0.8737738132476807, 1.612613558769226, -0.9068034291267395, 1.4914686679840088, -0.31312569975852966, 0.993029773235321, 0.1568583995103836, -0.5684024691581726, -1.1622307300567627, 0.046330153942108154, -0.688549816608429, 0.991273045539856, -1.0176701545715332, 0.5010995864868164, -0.2186318039894104, -0.2694917321205139, 0.5214523673057556, -0.6597891449928284, 0.4297420084476471, 0.3154534697532654, 0.7650720477104187, 0.4192257225513458, 0.5941323041915894, -0.6026943325996399, -0.8860428333282471, 1.8923232555389404, 0.6793313026428223, 1.039132833480835, 1.2357434034347534, -0.727418065071106, -0.1158057376742363, 1.245105504989624, 0.13261418044567108, 1.4725749492645264, -0.8824074268341064, 0.3329308032989502, -0.28351670503616333, 0.054685357958078384, 1.0470112562179565, -1.1660314798355103, 1.4609886407852173, -0.10804228484630585, -1.4834412336349487, -0.47403770685195923, -0.029248125851154327, -0.414119154214859, 0.4606182873249054, 1.086674690246582, -0.1656138151884079, -0.2428857833147049, 0.5974977612495422, 0.9490812420845032, 0.4435465633869171, 1.4134050607681274, 1.4035919904708862, -0.7342958450317383, -1.0189621448516846, 1.4635945558547974, 0.7245184779167175, -0.093169204890728, -0.33524423837661743, -0.2557508647441864, -0.353819340467453, 1.4451380968093872, 2.0871975421905518, 0.4005773067474365, -0.19381068646907806, -0.6270812749862671, -0.8711786270141602, 0.3465932309627533, 1.6830627918243408, 1.314028263092041, 1.0104615688323975, -0.5134865641593933, -0.15294523537158966, 0.01119782030582428, -0.12637466192245483, -0.23029477894306183, -0.3211139440536499, -0.5326395034790039, -0.9246577024459839, 0.7591793537139893, -0.03847847878932953, -2.3579342365264893, 1.3975034952163696, -0.07312226295471191, 1.01986563205719, -0.9307202100753784, 0.5064679384231567, -0.8072706460952759, 0.6444306969642639, 1.011432409286499, -0.26743239164352417, 0.4745532274246216, 0.5546784996986389, 0.20263154804706573, 0.255351185798645, -0.5660692453384399, 0.9701902866363525, 1.8057204484939575, -0.1884370595216751, -0.1636001467704773, -0.08004647493362427, 0.29147136211395264, 0.07883752137422562, -0.09706658869981766, 0.1655251830816269, -0.5201013684272766, 0.2810359001159668, -0.11610355228185654, 0.5030754804611206, 1.0441553592681885, 0.09165151417255402, 1.0705132484436035, 0.536196768283844, 0.708340048789978, -1.301222562789917, -1.04219388961792, -1.2710702419281006, 0.10720670223236084, -1.3831746578216553, -1.4391528367996216, -1.8636630773544312, 1.8270518779754639, -0.30904534459114075, 0.14067764580249786, -1.445742130279541, 1.4857592582702637, -1.0181165933609009, 0.3280782699584961, 2.0833537578582764, 0.6064198613166809, -0.6619279384613037, -1.2016855478286743, -0.6220189332962036, -0.3719865679740906, -1.2265030145645142, -1.6566015481948853, 2.2796123027801514, -0.965793788433075, -0.7187343835830688, 0.8248680233955383, -0.8639259934425354, 1.824005365371704, -0.8115220665931702, 0.728787362575531, 2.667757749557495, -2.137333393096924, -0.3098682761192322, 1.9227110147476196, -0.1622622311115265, -0.3833170533180237, 0.9108201265335083, -0.04783002659678459, -1.0691602230072021, -0.04281599819660187, -0.695496141910553, 0.09319429099559784, 0.3592051565647125, 1.763429880142212, 0.287971556186676, -0.3940353989601135, -0.8474740386009216, -0.44207465648651123, 0.32166460156440735, -0.3115871250629425, -0.007319317199289799, 1.1611814498901367, 1.1027164459228516, -0.08329879492521286, -1.805887222290039, 1.6079168319702148, 1.9118696451187134, 0.12787745893001556, -1.6336586475372314, -0.7660502195358276, 1.0423153638839722, 1.1921498775482178, 1.1656122207641602, 1.2909696102142334, 0.9055080413818359, -0.5143525004386902, 0.14765752851963043, 0.9300529956817627, -1.5444029569625854, -0.27029743790626526, -0.1713278740644455, 0.45241501927375793, -2.03473162651062, 1.3002742528915405, -0.9760280251502991, 0.5244084596633911, -0.3872455358505249, 1.2720139026641846, -0.33886435627937317, -1.150721549987793, 1.0836714506149292, 1.1074503660202026, 2.3963851928710938, -0.9835947155952454, 0.9487074017524719, 0.5815428495407104, 0.28786420822143555, 0.3035438358783722, -1.575236201286316, 1.0467438697814941, -0.22871142625808716, 0.7685230374336243, 1.0723190307617188, -1.512072205543518, 1.2241286039352417, -0.3819742798805237, 0.5575888752937317, -2.139538526535034, -0.20195287466049194, -0.5239827632904053, -0.7627149820327759, 0.8747752904891968, -0.9714066982269287, -1.2980616092681885, -0.1652684062719345, -0.4355635643005371, -0.5863510370254517, -0.7778312563896179, -0.6329174041748047, -0.6589335799217224, -1.241644263267517, 0.3864445388317108, -0.38238751888275146, -0.46496906876564026, 0.18899014592170715, 0.23049437999725342, -0.38645368814468384, 0.41086921095848083, 0.7031761407852173, -0.25159314274787903, -0.34525635838508606, 1.9310487508773804, 1.2910432815551758, -2.767681121826172, -1.1602507829666138, 0.5488259196281433, -1.8656420707702637, -0.1825464963912964, -0.048554614186286926, -0.08036837726831436, -1.1969146728515625, 0.6036810278892517, 1.1727771759033203, -2.217529058456421, -0.645488977432251, 0.29552552103996277, -0.9831621646881104, -0.1097654402256012, -1.0794577598571777, 0.7698801159858704, -1.0863898992538452, -0.8346379399299622, 0.35604411363601685, -1.314725637435913, 2.301436424255371, 0.7666006088256836, -0.8457192778587341, -1.3661943674087524, 0.7959074974060059, -0.8052543997764587, 0.5961592197418213, 1.1058276891708374, -1.6868271827697754, 0.9921892881393433, 1.2715461254119873, -0.08350484073162079, 0.007471640128642321, -0.5627371072769165, -0.704053521156311, -0.45256897807121277, 0.49216827750205994, -1.0187023878097534, -1.6321226358413696, 0.8488567471504211, -0.1930537074804306, 0.9201437830924988, 0.09437599033117294, 2.0333735942840576, -0.16408111155033112, 1.319048285484314, 1.1113852262496948, -1.7027119398117065, -0.3768381178379059, -1.1266196966171265, -1.407339334487915, -1.800558090209961, -0.19338490068912506, -1.6772063970565796, 1.067753553390503, 1.1013681888580322, 1.3227840662002563, -0.028680654242634773, -1.7224652767181396, 1.557903528213501, -0.4448202848434448, -0.8567668199539185, 1.168047308921814, 0.24078260362148285, 0.8022955656051636, 0.39135801792144775, -0.09136700630187988, 0.9795907735824585, 1.0133055448532104, 0.6442470550537109, 0.5556690692901611, 0.7818520069122314, 0.6992140412330627, -0.26011914014816284, 1.231691837310791, -0.9162654876708984, -1.4588391780853271, -0.14225199818611145, -1.781790852546692, 0.4386322796344757, -0.3082202970981598, 1.101883053779602, -2.856611490249634, 0.23552533984184265, 0.031488511711359024, -0.7474055290222168, -1.01375412940979, 1.3986371755599976, -0.26303184032440186, 0.16429255902767181, -1.1772128343582153, 0.48385173082351685, -1.0469481945037842, 0.7896607518196106, -1.1183756589889526, -0.4672519266605377, 0.6715512871742249, -1.259738802909851, -0.06791263073682785, 1.2629265785217285, 0.3153538405895233, 1.0474530458450317, -0.5276336073875427, -0.5783289074897766, 0.3724975287914276, -1.1733454465866089, 1.543582558631897, -0.6778109073638916, 1.673387050628662, 0.32987022399902344, 0.8445594310760498, 0.18533466756343842, 1.4578460454940796, 0.1497420370578766, -0.2015751451253891, -1.0338484048843384, 0.8785139322280884, -0.6483086943626404, 0.0874372124671936, -1.1204017400741577, -0.6151247024536133, -0.9423622488975525, -1.6407737731933594, -0.45378807187080383, 0.5926295518875122, -1.0070405006408691, 0.723628044128418, -2.296170711517334, -0.0071289995685219765, 0.5991363525390625, 0.6969776153564453, 2.5317938327789307, -0.0804692804813385, 0.35749348998069763, 0.11519285291433334, -1.052863597869873, 0.34892651438713074, -0.9342861771583557, -0.5112893581390381, 1.2913533449172974, -0.08437652140855789, 2.121255397796631, 1.2871204614639282, -0.785240650177002, 0.8316009044647217, -0.743108332157135, 1.322070837020874, 1.1125209331512451, -0.4638645052909851, 0.9024979472160339, -0.9691933989524841, -1.4596625566482544, -1.219720721244812, 2.4675309658050537, -1.0581756830215454, 0.40376198291778564, -2.683712959289551, -1.064399242401123, -0.5650650858879089, 1.5791302919387817, -0.04077261686325073, -1.8006681203842163, -0.8744012117385864, 0.04523424804210663, 0.251811146736145, 0.9674201011657715, 0.756610095500946, 0.22251874208450317, -0.7265446186065674, -1.427971363067627, 0.40291827917099, -0.32010015845298767, 0.2711177170276642, 0.3420027494430542, -0.34749042987823486, 0.5418103337287903, -2.9313206672668457, 1.0595877170562744, -1.0513743162155151, -1.112729549407959, -1.0297584533691406, 1.6664073467254639, -0.003308933461084962, 1.5113003253936768, 1.2786941528320312, 1.1791044473648071, 1.2788385152816772, 0.6585536599159241, 0.5876768827438354, -0.3658484220504761, -0.6148249506950378, 1.0580650568008423, 2.0020198822021484, 1.3735960721969604, 0.8352101445198059, -0.002391367917880416, -0.545090913772583, -1.7593954801559448, -0.31672680377960205, -0.8420753479003906, 1.8756327629089355, 0.33584165573120117, 0.5425822138786316, 0.9357538819313049, 0.6405004858970642, -0.898759663105011, 0.5506783723831177, -0.005436995532363653, -0.847824215888977, -0.8102406859397888, -0.17437618970870972, 0.14099033176898956, 0.4712406396865845, 0.8879580497741699, -1.283004641532898, 1.1460566520690918, 0.1054437905550003, 1.8953649997711182, 2.0392072200775146, 0.1394314169883728, 0.8565036654472351, 0.5750653147697449, 0.3110238313674927, 0.03588612750172615, -0.3433426022529602, 0.26392731070518494, -0.5531227588653564, 0.6333101987838745, 1.4135056734085083, 0.1458836942911148, 1.869039535522461, -0.3437519669532776, -0.18474598228931427, 0.46848222613334656, -0.8489392995834351, 0.0070634325966238976, -0.24012695252895355, -1.019174575805664, 0.8857039213180542, -0.8328759074211121, 0.6548163890838623, 1.2318074703216553, 0.3625587821006775, -2.6071949005126953, -0.7233144044876099, -2.383373737335205, 0.5445134043693542, -0.33414506912231445, 0.2804967164993286, -1.4498893022537231, -0.1893736869096756, -1.3304637670516968, 0.2382422536611557, -0.948834240436554, -1.0065146684646606, -0.2249847650527954, -0.22626011073589325, -0.307953417301178, 0.49355900287628174, -0.017775028944015503, 0.9956035614013672, 0.2371751368045807, 0.5502627491950989, -1.2706149816513062, -0.21749171614646912, 0.0765310525894165, 0.1403178721666336, 0.09138084948062897, -0.6163246035575867, 1.2829980850219727, -0.7423391938209534, 0.37511757016181946, 0.40507709980010986, 0.18973125517368317, 0.2977209985256195, 0.1642855405807495, 0.7585611939430237, 2.9062728881835938, 0.5040161609649658, 0.45518752932548523, 2.259064197540283, 0.6616004705429077, 2.230017900466919, 0.24669799208641052, -0.9724538326263428, 0.7632764577865601, 0.3128430247306824, -0.48712146282196045, -0.3303539454936981, 0.11801980435848236, 1.14680814743042, -1.5102860927581787, -0.8122624158859253, 2.181591749191284, -1.6513020992279053, -1.5320881605148315, -0.49317437410354614, -1.0956807136535645, 1.0448545217514038, -0.9211987853050232, -0.3735695779323578, 1.3928641080856323, -1.357654094696045, 0.3702179491519928, -2.693164348602295, -1.1031684875488281, -0.6236599683761597, -0.34130480885505676, -1.2215569019317627, -0.25100862979888916, 0.6289070844650269, 1.2591667175292969, -0.5466616153717041, -0.5691700577735901, 0.6528109908103943, -1.2097891569137573, 1.8203328847885132, -1.0354360342025757, 0.6498191356658936, -0.05560923367738724, -0.378337562084198, -1.4192787408828735, 0.5834847092628479, 1.433659315109253, -0.806423008441925, 0.8078320622444153, 0.8243227005004883, -1.4664005041122437, 2.380824565887451, 2.539109230041504, 1.9908241033554077, -0.8330209255218506, 0.3449389636516571, -1.2672922611236572, 0.5675937533378601, -0.7842413783073425, -1.0028769969940186, -0.023509617894887924, -0.43011999130249023, -0.366396427154541, 2.184605360031128, 0.6821394562721252, -0.9127985239028931, -1.0832356214523315, -0.4901379942893982, 0.6455276012420654, -1.48167884349823, 0.4561953842639923, -0.5643604397773743, 1.2531654834747314, 0.5867253541946411, -0.4821678102016449, -1.207633376121521, 0.5116010308265686, -0.43145546317100525, -0.21509720385074615, -0.7082425355911255, -0.15973427891731262, -0.9321808815002441, 0.25921568274497986, 0.4234023988246918, -0.8738941550254822, 0.5508305430412292, -2.0839805603027344, -1.5839718580245972, -0.19234952330589294, -0.5152528882026672, -1.2545534372329712, -1.0314480066299438, -0.12523174285888672, -1.158466100692749, -0.9155590534210205, -0.16041411459445953, 1.9293524026870728, -1.3394371271133423, 0.32053548097610474, -1.5208890438079834, 1.9635207653045654, 0.42652520537376404, 0.06270421296358109, -0.7197691202163696, 0.05849926546216011, 1.7224478721618652, 2.4685556888580322, 0.6007898449897766, -0.6397508382797241, -1.0408353805541992, -0.26470789313316345, -0.6969926953315735, -0.09288107603788376, -0.9996681809425354, 0.2355349361896515, -0.3748825192451477, -1.3499771356582642, -1.7936166524887085, 1.3509060144424438, 0.4651680588722229, 0.3246895372867584, 2.574707508087158, -0.3007352650165558, 1.8707386255264282, -1.1443047523498535, 0.7301696538925171, -0.023272031918168068, -1.0274732112884521, 0.7483125329017639, 0.08640272915363312, 1.8697279691696167, 0.14890159666538239, -0.420320987701416, -0.039791662245988846, 0.4302945137023926, 0.46835240721702576, 0.01460985653102398, -0.3463468849658966, 0.32849377393722534, 0.7709019184112549, 0.07362177968025208, -0.7910025715827942, -0.010261298157274723, -1.2433912754058838, -0.8992546796798706, -0.1715538650751114, 0.9631408452987671, -0.24160656332969666, 1.3323564529418945, -1.6604392528533936, -0.4026896059513092, -0.6708723902702332, 1.071943998336792, -0.19791144132614136, 1.2716395854949951, 0.4876629114151001, 0.05604417994618416, -1.4721993207931519, -1.4835271835327148, -0.02296164631843567, -0.44442540407180786, -0.1391783356666565, 0.04632514715194702, -1.7350497245788574, 1.6512707471847534, 0.34059372544288635, 1.1653308868408203, -0.8009776473045349, -1.2526044845581055, -1.1814459562301636, -0.6686230897903442, 0.7814923524856567, 0.6412168741226196, 1.3790972232818604, 0.6849550008773804, 0.8456576466560364, 1.399609088897705, 0.21469947695732117, -0.2268172800540924, 0.09837549924850464, -0.3347502052783966, -0.12183104455471039, -0.24065415561199188, 0.1608899086713791, 0.120479516685009, -0.14383141696453094, 1.3304533958435059, 0.4298301041126251, 1.8348129987716675, 0.4019843637943268, -1.6503980159759521, 1.3327500820159912, 1.4507579803466797, 0.9045165181159973, 1.2962439060211182, 0.08249682188034058, -0.606241762638092, -0.5722146034240723, 1.768515944480896, 0.1443084180355072, 0.3813089430332184, -0.12039604783058167, -0.24560467898845673, 2.706873655319214, -0.77720707654953, 0.04167779162526131, 1.1123969554901123, 0.21061496436595917, -0.6947557926177979, 0.3395612835884094, 0.706581175327301, 0.6405547261238098, -1.2828625440597534, 0.5582827925682068, 0.5082600712776184, 0.10218524932861328, 0.02310054376721382, -0.22496485710144043, -0.3829715847969055, 1.6038647890090942, 0.3723568320274353, -0.24059545993804932, -0.12880346179008484, -1.175896406173706, 0.33862510323524475, -1.161431908607483, -1.4801678657531738, 0.10325219482183456, -0.737096905708313, 0.2791346311569214, -2.2979471683502197, -0.46875664591789246, 1.9865144491195679, -0.35844600200653076, -0.5089349746704102, 0.9252551794052124, 0.14746086299419403, 0.19062314927577972, -0.32299795746803284, -0.4184110462665558, -1.2551909685134888, 2.2938857078552246, 1.6235836744308472, 0.3408648371696472, -0.6223602294921875, -0.7077314257621765, -0.7637214660644531, -0.22679482400417328, 0.6703956723213196, 0.6891216039657593, 0.282117635011673, -1.589621901512146, -0.37097012996673584, 0.6996645331382751, -1.501116156578064, 0.08312811702489853, -0.8179647326469421, -0.45402321219444275, -0.8639034628868103, -0.24406473338603973, 0.7857351899147034, 1.6256427764892578, -0.8436945676803589, -0.4393658936023712, 0.17018622159957886, 1.380067229270935, -1.616563320159912, 0.2752097249031067, -0.9185234904289246, -0.11438214778900146, 1.728890061378479, 0.9262374639511108, 0.7223699688911438, -0.4401733875274658, 1.818045735359192, 1.9451192617416382, -1.517004132270813, -0.5057533979415894, 0.6563842296600342, -0.11306288093328476, -0.8595195412635803, -0.5604023337364197, 0.5601698756217957, -0.649276077747345, 0.7749934792518616, -2.4637372493743896, -2.370943307876587, 0.20810313522815704, 1.069610595703125, 0.9105636477470398, 0.0036561053711920977, -0.14975984394550323, -1.8057886362075806, -0.3109385669231415, -0.48385652899742126, 1.3652973175048828, -1.0313373804092407, -0.7394313216209412, -0.6701041460037231, -1.898047685623169, -1.9971096515655518, 1.1911969184875488, -0.1474149525165558, 1.0490803718566895, -2.729616641998291, 0.07499511539936066, 1.179254412651062, -0.8889368772506714, -0.06422434747219086, 0.9288576245307922, 1.121456265449524, -1.270951509475708, -0.5621118545532227, 2.117859363555908, -0.531964898109436, -0.5021063685417175, -0.5074071288108826, -0.10203699767589569, 0.8011234998703003, -1.571531891822815, 2.6130423545837402, -0.4516910910606384, 0.27619996666908264, -0.19954556226730347, 0.25644034147262573, 0.07711292803287506, -0.5359945893287659, -0.6196883320808411, 0.6627836227416992, 0.3077106177806854, -0.9157797694206238, -2.0603702068328857, -0.3602582812309265, 0.16853521764278412, 0.9589832425117493, -0.7534578442573547, -0.3205021023750305, 0.6036524176597595, -0.09015888720750809, -0.15146297216415405, 1.634610652923584, 0.38423317670822144, -1.4474109411239624, 1.7335952520370483, -1.109392762184143, -0.5570315718650818, -0.40305742621421814, 0.9441964626312256, 0.1509527564048767, 1.1727149486541748, 0.11472135782241821, 0.26656728982925415, 0.6508673429489136, 0.8189384341239929, 0.05904676765203476, -0.3532123267650604, 1.3025319576263428, 0.48000696301460266, 1.2534164190292358, -0.4066060185432434, -2.292527675628662, 3.4123148918151855, -0.5583013296127319, 0.467116117477417, -0.7105022072792053, 0.4050848186016083, 1.5961827039718628, -0.6145563125610352, -0.5485668182373047, 0.577490508556366, -0.07829637080430984, 1.947551965713501, -0.3187420070171356, -1.841946005821228, -1.740655779838562, -0.25618281960487366, 0.21735893189907074, -0.16565954685211182, -0.4230492115020752, 1.1312036514282227, 0.23397667706012726, 0.8375938534736633, -0.6694262027740479, 1.0131498575210571, -1.0061945915222168, 1.034403681755066, -0.014186306856572628, -1.02220618724823, 0.6661491990089417, 0.31882065534591675, -1.4225693941116333, -0.8161489963531494, 0.8550350666046143, 0.2481137067079544, -0.44381311535835266, -0.37268975377082825, -0.8356979489326477, 0.9542925953865051, -0.9692131876945496, -1.7837365865707397, 0.012564164586365223, -1.4011090993881226, 1.9296029806137085, 0.7842428088188171, 0.7326080203056335, 0.16440856456756592, 0.9981335401535034, -0.22294588387012482, -0.585253894329071, 0.7378033995628357, -0.37277770042419434, -0.14598752558231354, 0.040648847818374634, 1.7848223447799683, 0.1276993751525879, 2.1725387573242188, 0.11298418790102005, -0.7694764137268066, 1.4284452199935913, 0.7284892201423645, -1.000430703163147, 0.21415509283542633, 1.6511765718460083, -0.06316062062978745, -0.4668548107147217, 0.8676909804344177, -0.17610470950603485, -0.724906325340271, 0.6124312877655029, 1.0574380159378052, -0.11007994413375854, 0.5679050087928772, -0.32723814249038696, -2.2202858924865723, -0.4578266739845276, -0.4098179340362549, 1.6938201189041138, 0.07805684953927994, 1.1267637014389038, 0.4863007664680481, 1.1656309366226196, -0.598197877407074, -0.9826244115829468, 0.47747546434402466, -1.1524090766906738, 1.4773372411727905, -0.9715239405632019, 1.1975016593933105, -0.29211845993995667, 0.5494993925094604, 0.5964291095733643, -1.5949310064315796, 0.24564731121063232, -2.0719919204711914, 1.5025724172592163, -0.29571741819381714, -0.294609010219574, -0.1537065953016281, 1.0784810781478882, -0.30810272693634033, -0.4631156027317047, -0.568890392780304, 0.7187281847000122, -0.8739988207817078, -0.33857259154319763, 0.14552117884159088, 0.5219683051109314, 0.4717884361743927, -0.9554004669189453, -1.1026523113250732, -0.28686341643333435, 0.3489055931568146, 2.337656021118164, 1.134084939956665, 0.33816587924957275, -0.7440569400787354, -0.8406355977058411, -0.6055103540420532, -0.5988406538963318, -1.1113266944885254, -0.30827099084854126, 0.03276896849274635, 0.7058015465736389, -0.12436941266059875, -0.8895174860954285, 0.8573065996170044, -1.844753384590149, 0.1651361584663391, 0.3079700767993927, 0.10408613830804825, -0.304453581571579, 0.03622988983988762, -0.06794402003288269, -0.8665658831596375, -0.40954893827438354, 0.4643568694591522, 0.7419325113296509, -1.316200613975525, -0.49000993371009827, 0.8929257392883301, -0.2011866271495819, -0.19269467890262604, -0.38501906394958496, 1.1069142818450928, 1.4654242992401123, 2.109503984451294, -1.237072467803955, -1.2616711854934692, -0.04045455530285835, 1.3066080808639526, 0.20528779923915863, -2.042231321334839, -2.3298888206481934, -0.14964671432971954, -1.5814828872680664, -1.102810025215149, 0.034758683294057846, 0.17216040194034576, -1.9631810188293457, -0.7527238726615906, 0.9660242795944214, -0.08131100982427597, -0.15635044872760773, 0.24351337552070618, 0.5878354907035828, 1.1991325616836548, -0.8013158440589905, 0.8551665544509888, 1.288844347000122, 0.8808268308639526, 1.3648914098739624, 0.2509821653366089, -2.2676236629486084, -0.5768359303474426, 0.7996662259101868, -0.20387019217014313, -2.565109968185425, -1.1863129138946533, -1.293123483657837, -0.6279308199882507, 0.5298404097557068, 0.6883419752120972, 0.16428892314434052, 0.4857388436794281, 1.4456284046173096, 0.31804540753364563, 0.020219890400767326, -1.6431524753570557, 1.4726741313934326, 0.3179585039615631, 1.0876110792160034, -1.4422872066497803, -0.7969668507575989, 0.10851453244686127, -1.5005797147750854, 2.2411820888519287, 0.35943639278411865, 0.03498987853527069, -0.844014048576355, 1.1206212043762207, -0.1817377656698227, 0.48157748579978943, 0.19959211349487305, -1.0350803136825562, -1.3236829042434692, 0.3145037591457367, 0.3621257543563843, -0.0048388405703008175, 0.8669946789741516, 1.4996259212493896, 2.616436004638672, -0.5955671072006226, -1.0606399774551392, -0.05703836306929588, 1.0776275396347046, 1.915046215057373, 1.4821559190750122, 1.7825539112091064, 1.4871907234191895, -0.10596339404582977, -0.6277347207069397, -0.6066933274269104, 0.0767076388001442, -0.8908308744430542, -2.0666725635528564, 0.4289485216140747, 0.4693131744861603, 0.04781190678477287, -0.8090876340866089, -1.0256669521331787, -0.8328477740287781, 1.829779863357544, 0.6079836487770081, -0.032740771770477295, 0.7535216808319092, 0.3094562590122223, -0.32960352301597595, 0.5671166777610779, 0.7889261841773987, 0.32924219965934753, -0.14299850165843964, 2.1978299617767334, -1.6192384958267212, 0.2028515636920929, 0.6765373945236206, -0.23415039479732513, 1.1046661138534546, 1.0450295209884644, -1.0522360801696777, 0.7555248737335205, -2.5486574172973633, -0.2238229364156723, -0.7876675724983215, -0.09565544128417969, -0.31497684121131897, -1.3981351852416992, -0.40891173481941223, -1.428792119026184, 0.07782641798257828, -0.3909763693809509, 0.4836280941963196, -1.7094846963882446, 0.8222242593765259, 0.5572905540466309, -1.2238727807998657, -0.9273296594619751, 1.213436484336853, -0.7695770263671875, 0.7406057715415955, 0.07075107842683792, 0.9336544871330261, -0.4000643491744995, -1.5400851964950562, -0.06487516313791275, 0.09499675035476685, 0.8410851359367371, -0.9657964110374451, -0.4272550046443939, -0.2620998024940491, -0.24089060723781586, 0.16194382309913635, 0.14686934649944305, 1.0283180475234985, -0.6503424644470215, 0.7672204375267029, 0.317352294921875, 0.6443566679954529, -0.7428578734397888, 1.0735360383987427, 0.16545945405960083, -1.2743511199951172, 2.1066994667053223, 0.11098508536815643, -0.08973588049411774, 0.5230947136878967, -1.8852583169937134, 0.5014067888259888, -1.4705277681350708, -1.6466598510742188, -0.19768008589744568, -0.9217662811279297, -0.11761130392551422, 0.5392274856567383, -0.07589038461446762, 0.0611959807574749, 0.3695867955684662, -0.44342052936553955, 0.6854764819145203, -0.07543441653251648, -0.4606954753398895, 0.5785716772079468, -0.34616824984550476, 1.0439555644989014, -0.06614848226308823, 0.5109833478927612, 0.7785614728927612, 0.1482572853565216, -0.6807369589805603, 1.0089467763900757, -0.2781570255756378, -1.893239140510559, 1.0544509887695312, 0.23793378472328186, 2.5107123851776123, 0.40407633781433105, 0.44720080494880676, -1.3384144306182861, -1.1902388334274292, -0.3478032648563385, -1.3274526596069336, 0.5170037746429443, -1.2648600339889526, 0.6555992960929871, 0.4790867865085602, 0.5153640508651733, 0.8239319324493408, -0.002514923457056284, -0.726477324962616, 0.1367531269788742, -0.4389498829841614, -0.8250018954277039, -0.7848206162452698, -0.13858433067798615, -0.37159958481788635, -0.3224887549877167, -1.1855424642562866, 1.4569801092147827, -0.5123635530471802, -0.6593229174613953, 1.1770968437194824, 1.3356291055679321, -0.13149110972881317, -1.001888394355774, 0.50467848777771, 0.7120872139930725, 1.6182509660720825, 1.4606330394744873, -1.9086567163467407, -1.6975359916687012, 0.6257149577140808, -0.30011388659477234, 0.09558514505624771, -2.4125466346740723, 0.30610692501068115, 0.5213682055473328, -1.1767386198043823, -1.1837475299835205, -1.1434094905853271, 1.0736702680587769, -0.09044773876667023, 1.2265454530715942, 0.8975168466567993, 0.2703746557235718, -0.259101539850235, 0.838130533695221, -0.7341996431350708, 0.1710178256034851, -1.793238639831543, -0.21957507729530334, 1.0416204929351807, 0.6286717653274536, -2.5641496181488037, -0.5905472040176392, 0.7531139850616455, 0.500639796257019, -0.4859657883644104, -0.10119839757680893, -0.05754700303077698, -1.4327495098114014, -0.33420372009277344, 1.7420905828475952, 0.08840809762477875, -1.4677430391311646, -0.05589261278510094, 0.24510115385055542, -0.48462870717048645, 1.350901484489441, 0.2678707242012024, 2.3338282108306885, -0.20066413283348083, -1.6388866901397705, -1.5455524921417236, 0.8368446826934814, -0.29482218623161316, 1.2557481527328491, -0.5115089416503906, 0.6073752045631409, -1.6824647188186646, 0.04552830010652542, -0.3905823230743408, -1.3231197595596313, -1.1207143068313599, 0.2698637545108795, 1.0851243734359741, -0.9088281393051147, -0.43221741914749146, -0.46741369366645813, 0.19095568358898163, -0.42124149203300476, -0.44718676805496216, 2.1691715717315674, -0.4261890947818756, 0.9437812566757202, -0.0017870372394099832, -0.06667893379926682, -0.36871469020843506, -0.8584339618682861, -0.5365592241287231, 0.8579835891723633, 1.3687776327133179, -0.28845617175102234, 2.3491687774658203, -0.31972137093544006, 0.22365286946296692, 0.4004463851451874, 1.112519383430481, 0.7998204231262207, 1.0111109018325806, 0.9062211513519287, -1.1875746250152588, -0.8004676103591919, 1.3736907243728638, 1.6460645198822021, 0.6431160569190979, 0.6061097383499146, 1.483250379562378, -0.9633086919784546, 1.0484330654144287, -1.1250061988830566, -1.4945939779281616, 0.1449354887008667, -1.1363805532455444, 2.1792898178100586, -0.0766635462641716, -1.6557108163833618, -0.7281584739685059, -3.4414639472961426, 1.4539581537246704, 2.9262077808380127, 0.6324422955513, -0.7552076578140259, -0.6641901731491089, -0.2984228730201721, -0.23394672572612762, 1.507651925086975, -0.06311062723398209, -0.6896935105323792, 1.4988235235214233, 1.003490924835205, 0.7700827121734619, 0.24390164017677307, 1.4233994483947754, 0.07319692522287369, 1.5206384658813477, -1.195302963256836, 0.5131784081459045, -0.6374373435974121, 0.2548367381095886, 0.014178337529301643, 1.623605489730835, 0.3145686388015747, -0.09766427427530289, 0.7265487313270569, 0.6556288003921509, 0.8622586131095886, 0.2448776811361313, -0.13587668538093567, -1.8017867803573608, -0.7253479361534119, -0.9627877473831177, -1.318171739578247, -0.4357829988002777, -1.1462295055389404, -0.622043251991272, 0.3023635149002075, 0.41070711612701416, -1.6785986423492432, -0.2436593770980835, 0.7063234448432922, 0.7115907073020935, -0.7950965166091919, 0.10685839504003525, 2.5808446407318115, -0.2132285088300705, -1.1054891347885132, -0.27656811475753784, 1.633559226989746, -0.4290984570980072, 0.1031927689909935, -0.5948722958564758, -0.11505745351314545, 0.4767809212207794, 0.07318462431430817, -1.1536632776260376, 1.6074756383895874, -1.3089890480041504, -0.04628922417759895, -0.6502938270568848, -0.687781810760498, -0.6558584570884705, -0.816937267780304, -0.7998380661010742, 0.043124422430992126, -1.6289684772491455, 1.4838167428970337, -1.2324421405792236, -1.5642437934875488, 1.1628215312957764, 1.908496618270874, -0.3676907420158386, 0.17043741047382355, 1.8721630573272705, -1.8755667209625244, 1.2257028818130493, -0.8710408210754395, 0.5059248805046082, -0.4624626636505127, 0.3645603656768799, -3.0218749046325684, 0.4080997407436371, -0.2143469601869583, -0.9412243962287903, -0.7308567762374878, -0.6508904099464417, -0.3115435838699341, 0.5373802781105042, 1.4210126399993896, 1.6361032724380493, 0.8409722447395325, 0.13889148831367493, -0.3929096460342407, -0.8445446491241455, 0.25849980115890503, -0.09692439436912537, 0.22963394224643707, 0.5007781982421875, 0.8148468136787415, 1.069831132888794, -0.16599276661872864, -1.415533423423767, -0.027845539152622223, -0.7207481265068054, 0.6766245365142822, -1.0876286029815674, 0.6696146130561829, -0.6386269927024841, 0.44247502088546753, -0.046061381697654724, 1.1488382816314697, -0.19218957424163818, -1.3090494871139526, -0.6049568057060242, -0.8576660752296448, 1.1114767789840698, -0.7345368266105652, 1.2542186975479126, -0.10233145952224731, -1.1761105060577393, -0.27566009759902954, 0.31377869844436646, -0.5684148073196411, -1.0199203491210938, -0.6673725247383118, 1.7926373481750488, 1.437034249305725, -0.09430185705423355, 1.3634587526321411, -0.9397156834602356, -0.25491559505462646, 0.6220956444740295, 0.028160080313682556, 0.7658818364143372, 0.8775128126144409, 0.16713374853134155, 0.8309894800186157, 1.5348867177963257, -1.3896235227584839, 0.02753535471856594, 1.2020095586776733, 0.5282038450241089, -0.7299059629440308, -0.9867939352989197, 0.4592365324497223, -0.4985537827014923, -0.03630993515253067, -0.034801363945007324, 0.5050481557846069, -0.3688690960407257, 0.734605610370636, -2.7353663444519043, 0.4034865200519562, -0.7177714705467224, -0.5045934915542603, -0.3721678555011749, 0.3438989818096161, 0.9405683875083923, 0.2961772084236145, 0.7114695906639099, -0.15536625683307648, -1.8793190717697144, 0.618698239326477, 0.1765572428703308, -0.31166931986808777, -0.6094690561294556, 0.012126131914556026, 0.5217315554618835, -1.4811545610427856, -0.3629440665245056, 0.466666579246521, 1.4246156215667725, 0.44918256998062134, -0.12531371414661407, -1.9468588829040527, -0.7145498991012573, -0.7058157324790955, -1.4115769863128662, 1.699196219444275, 0.08454254269599915, -3.156083106994629, -0.9014462232589722, -0.6397214531898499, -0.7194791436195374, -0.6340668201446533, -0.2547973394393921, 0.9238991737365723, 0.09784388542175293, -0.03962801769375801, 0.48051828145980835, -0.330258846282959, -0.6991368532180786, 0.38807055354118347, -1.3715946674346924, -0.3536718189716339, -0.5597518682479858, 0.9780803918838501, -0.9793884754180908, 0.205191969871521, 0.783358633518219, -0.5864206552505493, -0.45468178391456604, -0.6738813519477844, -1.0442173480987549, -1.1078599691390991, -1.2262767553329468, 1.696439266204834, 0.9988014698028564, 1.3766565322875977, 1.22294282913208, 1.614320158958435, 1.3888227939605713, 1.3191829919815063, 0.0514310859143734, -1.0662689208984375, -0.41341859102249146, -2.700493812561035, -0.974496603012085, -0.9339970946311951, -0.19144262373447418, 1.053716778755188, 0.2615833878517151, 1.030211091041565, -1.5348278284072876, 0.321473091840744, 1.2854944467544556, 0.7071738243103027, -1.2424432039260864, 0.5714728832244873, 0.26807478070259094, -1.3548922538757324, 1.958693504333496, -1.8723762035369873, -0.5800838470458984, 0.6226443648338318, -1.3915151357650757, 0.5983673334121704, -0.44047608971595764, -0.12278740853071213, -0.9397981762886047, -1.070652723312378, 0.285940945148468, -0.953457772731781, -0.5117756724357605, -0.2706562876701355, -0.9590490460395813, 0.12438155710697174, 1.313231348991394, -2.0931787490844727, 1.4761337041854858, -1.0901325941085815, 0.40886613726615906, 0.5621970891952515, -1.0203964710235596, -0.635917067527771, -1.256908893585205, -0.9342884421348572, -0.20643067359924316, -1.270870566368103, -2.234783172607422, -0.14614690840244293, 0.2713358402252197, -0.9136135578155518, 1.7996660470962524, -1.2264550924301147, 0.09708790481090546, 0.12485825270414352, -0.8088166117668152, 1.154811978340149, 0.47818079590797424, -0.23360945284366608, 0.060849033296108246, 1.0398216247558594, -0.26198697090148926, 1.139056921005249, 0.0964525118470192, 0.14262476563453674, 0.7808657884597778, -0.8153947591781616, 1.7695523500442505, -0.6893499493598938, -0.0877998098731041, -0.021174795925617218, 0.1712702214717865, 0.37405475974082947, -0.25363391637802124, -1.5888420343399048, -1.3066812753677368, 0.1562524288892746, 0.8166002035140991, 0.3636675477027893, -1.4679170846939087, 1.5496631860733032, 0.28757983446121216, 1.2174583673477173, 0.475269615650177, -0.20658418536186218, -1.2886444330215454, 1.5574396848678589, -0.16399841010570526, 1.9884438514709473, 0.45497316122055054, -1.783534049987793, -0.37073802947998047, -1.119563341140747, -1.157696008682251, -0.3800828754901886, -0.2531278133392334, 0.6257098913192749], [1.570296049118042, -0.5560978651046753, 0.051518701016902924, 1.4644652605056763, -0.7763977646827698, 0.22645682096481323, 1.0951014757156372, 0.844167172908783, 1.501548409461975, -1.7803032398223877, -2.5484306812286377, 0.031714294105768204, -0.4266878664493561, 0.3946512043476105, 0.9135860204696655, -1.2266480922698975, 1.110804557800293, -0.7932027578353882, 0.5372549295425415, 0.5824664831161499, 0.6928030252456665, -1.4170513153076172, -2.181328058242798, 0.38713327050209045, 1.1889094114303589, -0.9883447289466858, 0.8690778017044067, 1.7995857000350952, -0.0743321105837822, 0.9742284417152405, -1.2242671251296997, -2.5297367572784424, 0.9201791286468506, 0.22032149136066437, 1.145127773284912, -0.1643494963645935, 0.430526465177536, 2.5331764221191406, -1.1560027599334717, -1.4036777019500732, 1.7288810014724731, -0.8564209342002869, 0.5819128155708313, 0.7019355297088623, 0.2649131417274475, 0.35764065384864807, -0.10181048512458801, -0.8869650363922119, 1.0195435285568237, -2.097234010696411, 1.7667580842971802, 0.30129390954971313, -0.7948404550552368, -1.8040844202041626, 0.5178060531616211, 1.1973620653152466, 0.838422954082489, 0.5236167907714844, -0.9117808938026428, -0.7067555785179138, -2.1856555938720703, 0.23987218737602234, -0.37541303038597107, -0.5807082653045654, -0.4264819622039795, -1.7806768417358398, 0.5233718156814575, 0.33448904752731323, 0.8771051168441772, -0.25109919905662537, -0.7484286427497864, -0.9039900898933411, -1.918264389038086, -1.48005211353302, 1.0539308786392212, -1.2192384004592896, 1.2443445920944214, 1.2879716157913208, 0.2471672147512436, 0.8632183074951172, -1.2406643629074097, 0.4556376338005066, -1.135318398475647, 1.1749013662338257, -0.7256648540496826, -0.5208422541618347, 0.3866037428379059, 0.7796338796615601, -0.7991134524345398, 1.8318437337875366, 1.6578351259231567, 0.8668283820152283, 1.1397902965545654, -0.40697237849235535, -2.529655694961548, -0.2108428180217743, -2.65315580368042, 1.4671107530593872, -0.9067662358283997, 1.7808071374893188, -0.4112043082714081, -0.5456045269966125, 0.9625571966171265, -0.694810688495636, -1.0052707195281982, -1.7531306743621826, 1.4581092596054077, 0.9825966358184814, 0.6771277785301208, -0.016904449090361595, -0.07615301012992859, -1.061115026473999, 0.41824719309806824, 0.39020437002182007, -1.408835530281067, -0.3391841650009155, -1.3351517915725708, -2.079451560974121, -0.5723991394042969, 0.46431779861450195, 0.20388242602348328, -0.2671225070953369, -0.6137374639511108, -0.8009331822395325, -0.560075581073761, 0.01629868522286415, 1.3399227857589722, -1.2259700298309326, 1.580393671989441, 2.095107078552246, 0.5838435888290405, 0.4028424918651581, -1.5049242973327637, -0.0182146355509758, 0.5480146408081055, 0.7924508452415466, 1.6142354011535645, -0.2805626392364502, 1.6692198514938354, -0.42874088883399963, -0.909912645816803, 2.570651054382324, 0.2264919877052307, -0.9201053380966187, -1.3819568157196045, -0.7989873886108398, -0.26944902539253235, 0.26620230078697205, 0.758461594581604, 1.7160080671310425, -1.280943512916565, -1.6467316150665283, -0.6437814235687256, 0.6394237279891968, -0.4114968478679657, -0.807672381401062, -0.25684553384780884, 1.1455929279327393, 0.6478846073150635, 1.472367763519287, 0.465512752532959, -1.2770737409591675, -0.2802829444408417, -1.633263349533081, -1.5662531852722168, 0.09477529674768448, -1.7757186889648438, -2.2478127479553223, 0.34328514337539673, 0.11046433448791504, 0.14032214879989624, 1.0818052291870117, 0.23147691786289215, -1.0840370655059814, -0.09021808952093124, 0.5651488304138184, -1.9654853343963623, -1.0694371461868286, -1.5108728408813477, -1.1343497037887573, -1.513594627380371, 0.8894152045249939, -1.3512321710586548, -1.4218093156814575, -1.185180902481079, 0.09814770519733429, -0.13164517283439636, -0.09630691260099411, -0.34239843487739563, 0.6868821978569031, -0.7218858003616333, -0.6150830388069153, -1.58901846408844, 0.587925136089325, 0.4825383722782135, -1.1302844285964966, -1.1896166801452637, 0.4247751533985138, -0.9537158608436584, 0.01097091380506754, 1.0801169872283936, -1.2284305095672607, -0.3273056447505951, 1.1040947437286377, -0.09084044396877289, -0.1818597912788391, 0.28913745284080505, -1.3287632465362549, 0.8895276784896851, -0.06173359602689743, -0.1622820645570755, 1.3023959398269653, 0.5569954514503479, 0.8017142415046692, 0.7144762277603149, 0.06870941817760468, -0.3896732032299042, 0.4507442116737366, -0.5152081847190857, 1.2258594036102295, 0.5765032768249512, 0.161790668964386, 0.055734239518642426, 1.5696765184402466, -0.03503810241818428, 0.6915097236633301, 0.42273423075675964, 0.9831823706626892, -0.4267415404319763, -0.7377592325210571, -0.7536238431930542, 0.5965936183929443, 0.38032665848731995, -0.3968450427055359, -0.18676911294460297, 1.2645273208618164, -0.9507881999015808, 0.15907806158065796, -0.5920926332473755, -0.171543687582016, 1.0460506677627563, -0.38640397787094116, 0.31592926383018494, -1.1119894981384277, 0.24198994040489197, 0.5964126586914062, -1.975748062133789, -0.07704196870326996, 0.8704515099525452, 0.5312239527702332, 1.3282815217971802, -1.5592706203460693, 1.353764533996582, -0.4626578092575073, -0.8415949940681458, -0.7939307689666748, -0.9861096143722534, 0.7102879881858826, -0.1000543162226677, 1.4712092876434326, 1.8967355489730835, 2.4400439262390137, -1.00009286403656, -1.4399996995925903, 0.9179459810256958, 1.2608076333999634, -1.4532970190048218, -0.16564905643463135, 0.547336995601654, -1.6423009634017944, 1.2140780687332153, 2.667682647705078, 0.7611376643180847, 0.3573022782802582, 0.6713427901268005, -0.6964914202690125, 0.07836080342531204, -0.6449090838432312, -1.4891386032104492, 0.21091148257255554, 1.2513350248336792, 0.48529478907585144, -1.611352801322937, 2.297032117843628, 1.7756706476211548, -0.6591312885284424, 1.7096275091171265, -2.378439426422119, 0.7540137767791748, -0.4813706576824188, 0.20654305815696716, 0.4319290816783905, -1.409523367881775, 0.11732150614261627, -0.012097787111997604, 0.379462331533432, -0.06500231474637985, -1.812461495399475, 0.8866446018218994, 0.719052255153656, -2.1468582153320312, 1.9042325019836426, -0.11449795961380005, 0.4617244601249695, -0.7789331078529358, 0.19425570964813232, -0.27224311232566833, -0.29998093843460083, 0.039160989224910736, 0.22982971370220184, 0.024411000311374664, -0.0407540500164032, -0.3264753818511963, -0.3695999085903168, -0.4173997640609741, 0.4733518362045288, -0.4657082259654999, 0.5388257503509521, 0.37914159893989563, -0.05814872682094574, 1.2007410526275635, -0.48556241393089294, 0.6518716812133789, 0.6134611964225769, -0.15387289226055145, 0.5740712881088257, -0.4025005102157593, 1.3232802152633667, 0.7603299021720886, -0.8136739730834961, 0.25259578227996826, -0.3031867444515228, -0.2802911698818207, -0.22783263027668, -1.2195039987564087, -1.4222197532653809, 0.5581292510032654, -0.04037735238671303, 0.686715304851532, 1.5819028615951538, 0.7723298072814941, -0.64397132396698, 0.3984491229057312, 0.4267870783805847, -0.34168875217437744, -1.1519159078598022, 1.170061707496643, 1.2275676727294922, 0.11143968999385834, -0.14591529965400696, -1.147312879562378, -1.39315664768219, 0.27289143204689026, 1.1960771083831787, 1.4893908500671387, 0.7108557820320129, 0.02186095342040062, -0.5124841928482056, -0.8678469657897949, -0.15574464201927185, -0.015737341716885567, -0.30336159467697144, 0.5131279826164246, 0.5689161419868469, 0.17869995534420013, 0.40991854667663574, -0.30784252285957336, -0.3142053484916687, -0.7244563102722168, 1.0604963302612305, -1.638069987297058, -1.0747438669204712, 0.4150344431400299, 0.7758029103279114, -0.7164316177368164, 0.9320062398910522, 0.060705989599227905, 0.213626891374588, 1.2289403676986694, -1.5218441486358643, -0.3263312876224518, 0.6816492080688477, -0.910971462726593, 0.7140514254570007, -0.5551586151123047, -0.44005945324897766, 2.1119399070739746, 1.973145842552185, 2.0482289791107178, -0.8646848797798157, -0.20411522686481476, -1.2208248376846313, -0.41819265484809875, 0.9561249017715454, 1.1311932802200317, 0.25104621052742004, -0.4336896240711212, 0.017253853380680084, -0.6517638564109802, 0.12994660437107086, -0.66673344373703, 1.223235011100769, 1.1062991619110107, 1.0725152492523193, -2.651106357574463, -0.3032982051372528, 0.7234923243522644, -1.6354045867919922, 0.5740992426872253, 0.9590516686439514, -1.3239383697509766, -2.1647448539733887, -0.7723048329353333, -0.5227819085121155, 0.49557024240493774, 0.7407382726669312, 0.24162034690380096, -0.5899605751037598, -0.02773481421172619, 0.17675109207630157, 2.187002658843994, 0.6484808921813965, -0.6937018632888794, 0.6990976929664612, 1.126843810081482, -0.24676965177059174, 0.0650029256939888, 0.5427313446998596, 0.29476040601730347, -0.6091870665550232, -0.4898255169391632, 1.8738998174667358, -1.3898621797561646, 1.4803624153137207, -0.8487991094589233, 0.5645252466201782, 0.03609749302268028, 1.4060866832733154, -1.080230712890625, 1.1044126749038696, -0.6222359538078308, 0.5874715447425842, -1.3527330160140991, 1.08790922164917, -0.36914992332458496, -0.2695826292037964, 1.6431334018707275, 0.6126826405525208, -0.3386955261230469, -0.7971678972244263, 0.031224939972162247, -0.6507500410079956, -0.33644720911979675, -0.11342588812112808, 0.1893659383058548, -0.7318815588951111, -0.09789009392261505, 0.6236237287521362, -0.4428679347038269, -0.6211186647415161, 0.8763877153396606, 0.7313156127929688, -0.05000017583370209, 0.4937003254890442, -1.0734180212020874, -0.012179957702755928, -1.5867319107055664, 0.15581899881362915, -0.22150929272174835, 1.2658040523529053, 0.6750698089599609, 0.1027148887515068, 1.6729650497436523, -1.743143081665039, -0.38551613688468933, 0.9552296996116638, -1.2557566165924072, -0.40937644243240356, -1.7598776817321777, 0.6076018214225769, -0.8145566582679749, -1.5093708038330078, -0.03198764845728874, -1.175303339958191, 0.8988187313079834, 0.468168169260025, -0.4494439959526062, -0.03158343583345413, 0.7110010981559753, 2.3512041568756104, 0.3149067461490631, -0.6745606064796448, -0.05329965800046921, 0.7482693195343018, -0.5459433794021606, -1.0687228441238403, -0.3187401294708252, 0.22075670957565308, -0.6901143789291382, -0.8169319033622742, 2.0351388454437256, 1.8123427629470825, -0.5585792660713196, -0.2509465217590332, 0.4168162941932678, -0.8980503082275391, 0.37135806679725647, -1.1144880056381226, -0.44879111647605896, -2.883528709411621, 2.792537212371826, 0.1331232488155365, 0.046434320509433746, -1.221044659614563, 0.08884623646736145, -0.5139676928520203, 1.01068115234375, 0.1211920976638794, 0.63785719871521, -0.6616020202636719, 0.6692846417427063, -1.4027515649795532, -0.4464547336101532, -0.2913404405117035, 0.21950630843639374, -0.3498900234699249, -1.37856125831604, -0.04077048599720001, -1.6491824388504028, 0.4709840416908264, 1.1615067720413208, -0.5221489071846008, -0.9765242338180542, -1.4315909147262573, -0.28754934668540955, -0.6403954029083252, -0.06881330907344818, -0.18083693087100983, -0.23671260476112366, -0.15377168357372284, 0.46116504073143005, 0.9684010744094849, -1.2424190044403076, -0.47589412331581116, 1.0735622644424438, -0.26355263590812683, -0.2637638747692108, 1.246267318725586, -0.03459632024168968, -0.09549888223409653, -0.9002309441566467, 0.5007297396659851, -0.13352839648723602, -0.5019782781600952, 0.35016846656799316, 1.404880404472351, -0.4567639231681824, 0.3168155550956726, -0.6608524322509766, 0.06583667546510696, -0.3967093825340271, 0.07691667973995209, -0.27224674820899963, 2.0916941165924072, 0.04904831573367119, 1.870676040649414, -1.6445198059082031, 0.24685099720954895, -0.9770547151565552, 0.444275438785553, -1.9810229539871216, -1.5971927642822266, -0.22510680556297302, -0.4177635908126831, -0.1788611263036728, 1.3096787929534912, -2.049546003341675, -1.2551133632659912, -1.9224331378936768, 1.0269365310668945, 0.23335614800453186, -0.8393509984016418, 2.3282508850097656, 0.9807236790657043, 0.8907614350318909, -0.07489783316850662, -0.9250689744949341, 0.4784506559371948, 0.8877821564674377, 0.9609348773956299, 0.17293018102645874, -1.218907117843628, 0.6473337411880493, 0.4728531241416931, 0.3458276391029358, -0.9883437156677246, 0.5787864923477173, 0.6743637919425964, 0.8627797961235046, -0.5773715972900391, -0.4272199869155884, -0.4654795825481415, 1.3244001865386963, 1.0130196809768677, -0.062031738460063934, 0.08485078811645508, -0.17059564590454102, 0.5467352867126465, -0.9485414624214172, -0.4227437674999237, -1.9422920942306519, 0.8552483916282654, 0.7710212469100952, -0.7463023066520691, 1.0531419515609741, 1.0366543531417847, -1.3828151226043701, 0.5802563428878784, 1.342319369316101, 0.3584554195404053, 0.9000576138496399, -0.5195040702819824, 1.8701056241989136, 1.601969838142395, -1.1523483991622925, -0.8933002948760986, 0.13164789974689484, -0.530426025390625, 1.6479322910308838, -1.4199248552322388, -0.31329962611198425, 0.7792261838912964, -0.4526066482067108, 0.1058046892285347, -0.3349378705024719, -0.3519324064254761, 0.9346946477890015, -0.19527418911457062, -0.5409916639328003, 1.9777652025222778, 1.2546318769454956, 2.1364786624908447, 0.023932084441184998, 1.1831251382827759, -0.7533717751502991, -0.8443590402603149, 0.004523384850472212, 0.5971932411193848, -0.2197258025407791, 0.3479354977607727, 1.1195178031921387, -2.2132532596588135, 1.2599748373031616, 0.023893704637885094, 0.24818289279937744, -0.6943219304084778, 1.4714829921722412, 0.12598609924316406, -0.618539035320282, -0.5772801637649536, -0.6567994356155396, -0.8494839072227478, 0.06920746713876724, -0.8601272106170654, 0.5289168357849121, 0.11819393932819366, 0.8437390923500061, 0.003905319608747959, -1.2759608030319214, 2.220349073410034, -0.28226974606513977, 1.3430438041687012, 0.41644471883773804, 0.007696027867496014, 0.6187416315078735, -0.26056671142578125, -1.9817376136779785, -0.2509341537952423, 1.967386245727539, -1.7200742959976196, 1.2672630548477173, -0.36942893266677856, -1.3863314390182495, 0.30060863494873047, 0.3531067967414856, -2.3275532722473145, -0.9034278988838196, -0.37703338265419006, -0.16618666052818298, -0.9913232326507568, -0.8698661923408508, 0.4818018078804016, 0.3185892105102539, 0.6638206243515015, 0.8831542134284973, -0.36012378334999084, 1.2934824228286743, -1.0842050313949585, 0.9222283959388733, 0.22895751893520355, -0.7413803339004517, 0.31449010968208313, 0.2221778780221939, 2.1012184619903564, -0.27682918310165405, -0.3562796115875244, 0.608936607837677, -0.1948556900024414, 0.6194698810577393, 1.2884210348129272, -2.232149600982666, -0.8363227844238281, -0.13075722754001617, 0.6515058875083923, -0.788017749786377, -1.4499799013137817, -2.7848763465881348, -1.157766580581665, -0.25821271538734436, 1.2264505624771118, 0.8129043579101562, -0.659777045249939, 0.632865309715271, 1.5887669324874878, -0.22570684552192688, 0.34226498007774353, 0.2789766192436218, -0.08363552391529083, -0.030684838071465492, -1.9992091655731201, 1.5750235319137573, -1.2534486055374146, 0.22339539229869843, 0.5826342701911926, 1.4199663400650024, -0.5189005136489868, 0.8913383483886719, 0.326349675655365, -0.03344245254993439, -1.5884613990783691, 0.7160859704017639, -1.6318669319152832, -0.9348002076148987, -0.12315992265939713, -1.7251490354537964, -0.08107897639274597, -0.5026748776435852, -0.11808709055185318, -0.2782551050186157, -1.925612211227417, 0.3981209993362427, 0.35461536049842834, -0.7328284382820129, 0.7765660881996155, -0.17391958832740784, 1.1841193437576294, -1.277955412864685, -1.8163273334503174, -0.5689924359321594, 2.37198805809021, 0.0960521250963211, 0.6794158816337585, 0.22469089925289154, -0.0584590807557106, -0.15765824913978577, 0.4431832730770111, 0.31945085525512695, -1.1497575044631958, -0.9656638503074646, 1.0735011100769043, 0.30165985226631165, -1.721998929977417, 0.3620430529117584, -0.9631847143173218, 1.7483556270599365, 0.37364518642425537, -0.6867510080337524, -1.6104036569595337, 1.4440302848815918, 1.3295890092849731, 1.500906229019165, -0.10849695652723312, 0.5353899002075195, -1.0054305791854858, 1.7034982442855835, 0.48628121614456177, 0.543773353099823, 0.03523574024438858, -2.5980591773986816, -0.7308408617973328, 1.084360957145691, 1.4979424476623535, -1.1884236335754395, 1.6347194910049438, -0.9732756018638611, 0.8190481066703796, -0.5977665185928345, -0.7278220653533936, -1.476348638534546, 1.8611637353897095, -0.7953505516052246, -0.02549569308757782, 0.5421851277351379, 0.028654756024479866, 0.510793924331665, -0.6523560285568237, 0.1824444979429245, -0.07457537949085236, 0.09609740972518921, -1.0081348419189453, 1.6859294176101685, 0.14159296452999115, 0.16337548196315765, 0.605465292930603, -0.30122366547584534, -0.698660671710968, 0.44362393021583557, 0.07708170264959335, -0.18109454214572906, 1.7433980703353882, -1.5621261596679688, -1.655696153640747, 0.886252760887146, -0.42825788259506226, -0.11893356591463089, 0.08499189466238022, -0.07069215178489685, -0.7614617943763733, 0.03738207370042801, 2.3658463954925537, 0.28963908553123474, 0.9042553901672363, 1.3412790298461914, -1.4594008922576904, 1.5008877515792847, -1.2513453960418701, 0.05801219493150711, 0.8287673592567444, 0.3635902404785156, -0.8280224800109863, 0.7047445178031921, 2.210928201675415, 0.621537983417511, 0.41948843002319336, -0.7922986745834351, 0.030881326645612717, 0.34892401099205017, 1.323033332824707, 0.4177183508872986, -1.0256479978561401, 0.2517041265964508, 1.2648210525512695, -0.5372190475463867, 0.042401302605867386, -0.2916255593299866, 0.26703333854675293, -0.23901066184043884, -0.5740927457809448, -1.5806639194488525, 0.6536623239517212, 0.24266988039016724, -1.0687862634658813, -1.616373062133789, 0.3100840449333191, 0.47938838601112366, -0.6510583162307739, -1.131166934967041, 0.7661281228065491, 1.5733168125152588, -1.8082821369171143, 0.40956106781959534, 0.7110571265220642, 0.8939944505691528, 0.711408257484436, -1.5598310232162476, 1.1568282842636108, -0.9069172739982605, -2.655886173248291, -1.189552903175354, 0.948412299156189, 0.08702097833156586, 0.39227384328842163, 0.3525826632976532, -0.7333253026008606, 0.3110048472881317, -0.1531791388988495, 0.9781601428985596, -0.9033652544021606, 0.910400390625, 0.036194074898958206, -0.2271580845117569, 0.601386308670044, 0.42557454109191895, -1.022400140762329, -0.3081153631210327, -0.025348497554659843, -0.3290543258190155, -0.733348548412323, -0.6285293698310852, -0.09333505481481552, -0.46319612860679626, 2.007920026779175, -0.3171062767505646, 0.5988923907279968, -0.7503212094306946, 0.04999619349837303, -0.5214954614639282, -0.7689312100410461, -0.29968225955963135, -0.847162127494812, -1.169934868812561, -0.49404600262641907, -2.427643299102783, -0.2925514876842499, 1.0574840307235718, -1.0944327116012573, 0.821573793888092, 1.3986103534698486, 0.8329327702522278, 0.10276714712381363, 1.275671124458313, 0.13497211039066315, -1.6701176166534424, -0.20922881364822388, -0.2018006592988968, 0.4698654115200043, -1.4488309621810913, -0.7000150680541992, -1.0499578714370728, 0.6225727796554565, 0.37501925230026245, 0.9165008068084717, 1.2071142196655273, 0.2930399775505066, 1.171433925628662, 1.3730944395065308, 0.5296657085418701, -0.9866903424263, 0.6618480086326599, 0.4039987623691559, -1.165218472480774, -0.8890146017074585, -1.6038334369659424, -0.6811988353729248, 1.9159871339797974, -0.6443542242050171, -0.9738793969154358, -1.8098911046981812, 0.00893403496593237, -0.5831913352012634, 1.751678466796875, -0.6460193395614624, -0.3617931604385376, 1.605252981185913, 1.1744325160980225, -1.0170187950134277, -0.36575332283973694, 0.5510126948356628, -0.7930451035499573, 0.07751326262950897, 0.3736400604248047, 0.5324028134346008, -0.19566203653812408, 0.21500153839588165, 0.29437705874443054, 0.14314605295658112, -0.4468614161014557, 1.9455147981643677, 0.4287523031234741, -0.8572399616241455, 0.4694754481315613, -0.20043720304965973, 0.05153788626194, 0.18711420893669128, 1.9256612062454224, -0.07484693080186844, 0.12092660367488861, 0.13167661428451538, 0.008389653638005257, 0.2922186851501465, 0.3017369508743286, -0.725626528263092, 0.9331875443458557, 2.2677032947540283, 0.1512998342514038, -1.3456814289093018, 0.47346732020378113, -0.19237740337848663, 0.9766160845756531, 0.6837444305419922, -0.6247971653938293, 1.551517128944397, -0.29127463698387146, -0.9215173721313477, 0.10476308315992355, -1.3005386590957642, -2.2759323120117188, 1.1429765224456787, -1.2773857116699219, -1.0162338018417358, -0.9894539713859558, 0.5611132979393005, 0.16280962526798248, -0.26081058382987976, -0.3043130338191986, -0.599757730960846, -1.1499930620193481, 1.4990299940109253, -0.707977294921875, -0.8732896447181702, 0.12546904385089874, 2.4187910556793213, -0.5960243344306946, 0.32598280906677246, 0.5490704774856567, -1.1036903858184814, 1.1574219465255737, -0.46763724088668823, 0.7905095815658569, -1.4182419776916504, -2.2258083820343018, 0.7916079163551331, -0.971015989780426, -0.8811855912208557, -0.3356533944606781, -1.1975696086883545, 2.0749576091766357, 0.8669931888580322, 0.45256227254867554, -1.1953353881835938, -0.43989697098731995, -0.8519033789634705, -0.2786923348903656, 0.8507711887359619, -0.21400700509548187, -0.8442655205726624, -1.236655354499817, 0.1217375099658966, 0.16277571022510529, -1.4901772737503052, 0.7581027746200562, -0.19527241587638855, 1.0973471403121948, -0.7971606850624084, 1.6570242643356323, -0.44610702991485596, -1.4134159088134766, 0.19870705902576447, -1.492329716682434, 0.4419489800930023, -0.41192376613616943, -0.8960281610488892, -0.9317274689674377, 0.21514977514743805, -0.14302174746990204, 0.026376087218523026, -0.24124915897846222, -1.235119104385376, -1.381584644317627, 0.10475318133831024, 0.28091558814048767, 0.42359524965286255, -0.9750295281410217, 1.2181822061538696, 0.2928358316421509, 0.5808619260787964, -1.519182801246643, -0.53653883934021, 0.8850663304328918, -2.015702486038208, -0.9650530815124512, -1.3534111976623535, -0.6813311576843262, 0.009921266697347164, 1.537976861000061, -0.4662782847881317, -1.278371810913086, 0.2015853375196457, 1.674392819404602, -1.2562299966812134, -0.47634804248809814, -0.4924352467060089, -1.27103590965271, 1.8118973970413208, -1.3902063369750977, -0.18837286531925201, -0.9816763401031494, -0.03519260510802269, 0.006161702796816826, 0.0688813254237175, 0.04190131649374962, -0.1631765216588974, 0.5090900659561157, 0.4508444368839264, 0.7584953904151917, 0.24139940738677979, -1.1661127805709839, -0.23422737419605255, -0.4856615364551544, -0.6486321091651917, -0.8535739779472351, -0.8270182013511658, -2.135530710220337, 2.550839424133301, 0.5137954354286194, 0.06814483553171158, 0.9720386266708374, 0.9412580132484436, -2.7928359508514404, 0.8305160999298096, -1.3872884511947632, 0.4152297377586365, 0.6525433659553528, -0.5521042346954346, 0.34056925773620605, -0.06174541637301445, 0.07028033584356308, -1.6939921379089355, 0.11406921595335007, -0.5027085542678833, -3.2707011699676514, -2.0941250324249268, 1.222949504852295, 1.1025969982147217, -0.2997075915336609, 0.38093703985214233, -0.9999643564224243, -0.5511507391929626, 0.8852744698524475, 1.2698763608932495, 0.6089780926704407, -0.5653830766677856, 1.9382492303848267, -0.7198821306228638, 0.913364589214325, 1.7266372442245483, -0.17587929964065552, -0.4419582486152649, 0.011414187029004097, 1.1986243724822998, -1.4357163906097412, 0.6093676686286926, -1.5756142139434814, -0.0322883240878582, -0.02142954431474209, -1.95219087600708, 0.6092529892921448, 1.7363593578338623, 1.2003308534622192, 2.5146703720092773, -0.8894347548484802, -0.3275081515312195, -1.0051770210266113, 0.10610784590244293, -0.5793353915214539, 1.4166069030761719, -0.6560361981391907, 0.04789767041802406, 0.7616948485374451, -0.9849637746810913, -0.48929640650749207, -1.205152153968811, 0.1420735865831375, -0.10650037229061127, -0.012216072529554367, -0.9845789670944214, 0.8173524141311646, 1.1017141342163086, -0.2106422483921051, -0.8393668532371521, -0.016010968014597893, -0.16719481348991394, 1.2683888673782349, -0.06312083452939987, -0.019229812547564507, -0.02000683918595314, -0.6530421376228333, 0.007552471477538347, 0.39878836274147034, 0.9224133491516113, 1.1129944324493408, -1.4164037704467773, 0.6933364272117615, -0.17622345685958862, 1.6920069456100464, -1.0457898378372192, -2.4708292484283447, 0.3073570430278778, -0.018552647903561592, -0.6106751561164856, -0.531070351600647, -0.7391493916511536, -0.09007710218429565, 0.15780414640903473, -0.472243070602417, -0.46746137738227844, 0.6947544813156128, -0.5455945730209351, -0.29995137453079224, 1.645801067352295, 0.966529905796051, 0.0710623636841774, 1.1189597845077515, -0.7681159973144531, -0.12638646364212036, 0.13873039186000824, -2.0744426250457764, -0.9687396883964539, -1.4182170629501343, 0.21065925061702728, 0.5773948431015015, 1.4800742864608765, 0.4275098443031311, 0.7520021796226501, 0.695331871509552, 1.6230299472808838, 0.3162863552570343, 0.0014505140716210008, -0.05852580815553665, 2.068661689758301, -1.288988709449768, 1.10693359375, 0.9769005179405212, -0.36869779229164124, -1.0364539623260498, -0.8843043446540833, 1.3968162536621094, 1.6108022928237915, 0.34118619561195374, 0.32934120297431946, 1.106508731842041, 0.6544339060783386, -0.6988745331764221, -0.5393797159194946, -0.14341787993907928, -1.3495087623596191, -0.03443131968379021, 0.02882179245352745, 0.612379252910614, -0.8385955095291138, -0.27905330061912537, -0.11616851389408112, 1.903241753578186, -0.5969874262809753, -1.6454458236694336, -0.9868807196617126, 0.3236153721809387, -1.2565401792526245, 0.4867349863052368, -0.5421774983406067, -1.9275805950164795, 0.45336657762527466, -1.501204490661621, 1.110711932182312, -1.0136570930480957, 0.12525296211242676, -0.6976339221000671, 0.2527351379394531, 0.4938708543777466, -0.2306281477212906, -0.33546170592308044, 0.8494454026222229, 0.7634371519088745, -1.3301475048065186, -1.3795292377471924, 0.6413091421127319, -0.006078016012907028, -0.5080218315124512, 0.2628307342529297, -1.9041942358016968, 0.05524958297610283, -0.27216246724128723, -0.6827241778373718, -0.30718904733657837, -0.25611618161201477, 1.6492791175842285, 1.1297791004180908, -0.22511063516139984, 1.0260530710220337, 0.08699340373277664, -0.761709451675415, 1.063704252243042, 0.2989329695701599, -0.31407129764556885, 1.9704256057739258, 1.7019685506820679, 1.5896718502044678, 1.1874933242797852, 0.026784226298332214, 1.6924861669540405, -0.6056559085845947, 0.3250786066055298, -0.8051551580429077, 0.47852420806884766, -1.1634788513183594, 2.72729754447937, 0.763667106628418, 0.2223270684480667, 0.40438544750213623, -0.5056801438331604, 0.2399255335330963, 0.23554371297359467, -0.6355903148651123, -0.9743186831474304, 1.6834211349487305, 0.4143744111061096, 1.2517309188842773, 0.07441919296979904, 1.490372657775879, -0.32276901602745056, -0.1351640522480011, -0.065603107213974, 0.7057967185974121, 0.49011075496673584, 0.013171081431210041, 0.6040867567062378, -0.09618601202964783, 1.084963321685791, -0.3416754901409149, -0.40245044231414795, -1.199679970741272, 1.774356484413147, -0.8053852319717407, -0.597545862197876, -1.1657298803329468, -0.2667683959007263, 0.6293773651123047, -0.18653404712677002, 1.9991291761398315, -0.568252444267273, -1.0150245428085327, -0.5842394828796387, 0.7351705431938171, 1.0528450012207031, 1.7272911071777344, -0.05348280072212219, -0.2953333556652069, -0.02632269263267517, 0.38166293501853943, -1.2821677923202515, -1.030288577079773, 0.42460867762565613, -0.5798379778862, -1.8210376501083374, 0.8786439895629883, -0.7870298624038696, 0.14836007356643677, 0.45788368582725525, -1.15988028049469, 1.2983356714248657, -1.0119552612304688, 0.8692296147346497, -1.4405338764190674, -1.0147945880889893, 1.3294599056243896, -0.16309866309165955, 0.45689499378204346, -0.28992852568626404, -1.9037467241287231, 0.5548738837242126, 0.06985427439212799, -1.8491299152374268, -1.9340041875839233, 0.5347850918769836, -0.22692640125751495, -0.504673182964325, 0.8374954462051392, -0.185263529419899, 0.588335394859314, -1.340914249420166, -0.9524668455123901, -0.9874972105026245, 0.22437313199043274, -0.4927632808685303, 1.0203967094421387, 0.3206242024898529, -0.5687857866287231, 0.5145434737205505, 1.1259199380874634, -2.0843863487243652, -1.957011103630066, 0.21760372817516327, -0.2585752606391907, 0.4721464514732361, -1.219297170639038, 0.4991428256034851, 0.40186670422554016, -1.1176207065582275, 2.0194618701934814, 0.05228515341877937, 0.6818576455116272, -1.21858811378479, -0.6475342512130737, -0.030356084927916527, 1.1698262691497803, -1.8278255462646484, -0.592322587966919, -0.23332734405994415, -0.7300803065299988, 0.417758971452713, -0.28926506638526917, 0.2850646674633026, -1.0219095945358276, -1.521803855895996, 0.562147855758667, 0.22345800697803497, -1.4938970804214478, -0.7560986876487732, -0.25653550028800964, -0.22040902078151703, -0.12425623089075089, -1.1891484260559082, 0.38957199454307556, 0.487865149974823, 2.02341365814209, -0.3570849597454071, -1.9358946084976196, 0.7741852402687073, 0.8483986258506775, 0.8276845812797546, -0.5025049448013306, -0.14258190989494324, -0.17800013720989227, -0.13984248042106628, -0.658958911895752, 0.6495478749275208, 0.25854846835136414, 1.143884301185608, 0.10849414765834808, -0.3195191025733948, 0.9196382761001587, -1.458985447883606, 0.2820736765861511, -0.07000325620174408, -1.5078626871109009, -0.8481171727180481, -0.7145698070526123, -0.7453242540359497, -0.47829726338386536, 1.5451356172561646, 0.5543113350868225, 0.10056596249341965, -1.5531753301620483, -1.613326072692871, 1.37253737449646, -0.03963352367281914, -1.1516112089157104, 0.7624281048774719, 0.6131635308265686, -0.844684898853302, -0.7528887391090393, -0.38221704959869385, -0.46405860781669617, 1.6413922309875488, -1.5245906114578247, 0.2022366225719452, -0.8237506151199341, -1.5209877490997314, -1.4473600387573242, 0.6592951416969299, 0.3090197741985321, 0.33505553007125854, -1.703550100326538, -0.14456021785736084, -1.0666338205337524, 1.1860915422439575, -0.8678868412971497, -0.40877217054367065, 0.8286117315292358, 0.5615593791007996, 0.1128239706158638, 0.9230523109436035, 0.23450659215450287, -0.5945033431053162, 0.15307864546775818, -0.17138849198818207, -0.8179791569709778, -0.8232437372207642, 1.4312026500701904, -1.2440515756607056, -2.418501138687134, -0.6990485191345215, -0.7287991642951965, -1.6060482263565063, -0.853395938873291, -0.6662248373031616, 0.4312129318714142, 1.1805306673049927, -0.851018488407135, 0.8688064217567444, 1.194875955581665, 1.4709970951080322, 0.4642563462257385, 0.9386016726493835, 0.12636157870292664, -0.1656825691461563, -1.893770694732666, 0.2391837239265442, 0.5578635334968567, -0.07662045955657959, 2.3526766300201416, 0.26694878935813904, 0.7612957954406738, 0.2385992556810379, 0.14932431280612946, 1.009825587272644, 0.631645917892456, -0.1973726451396942, 0.6783617734909058, 0.07575488090515137, -0.7293532490730286, -0.7266444563865662, -0.9592899680137634, -0.7238327860832214, 1.7066465616226196, -1.0510072708129883, -2.060107707977295, 1.8048763275146484, 0.8871620297431946, -0.7382245659828186, -1.5700575113296509, -0.676385760307312, -1.00993013381958, -1.312497615814209, -0.6945138573646545, -0.6468108892440796, -0.0403667613863945, 1.5055913925170898, 0.5087373852729797, -1.9008617401123047, 0.9311656355857849, 0.7953787446022034, -0.6998544335365295, -0.7396212816238403, 1.4186336994171143, -0.5313606262207031, 1.4396069049835205, 1.0425338745117188, 0.19584473967552185, -0.42200660705566406, -0.4705694913864136, -1.3050819635391235, -0.7030340433120728, 1.6130925416946411, -0.8829421401023865, 0.40586698055267334, -0.20022878050804138, -0.2389877587556839, -0.5316148996353149, -0.056803952902555466, -0.681900680065155, -1.2477648258209229, 0.8184161186218262, 0.03729958087205887, 0.30304425954818726, 0.3803271949291229, -1.240121841430664, -0.9400814771652222, -0.15027646720409393, 0.18150736391544342, 0.11465460062026978, 0.44064250588417053, 1.7961176633834839, -0.004630897659808397, -1.6104416847229004, 0.9718458652496338, 0.17302998900413513, -0.4228435456752777, -0.6797916889190674, 1.5566297769546509, 0.21030914783477783, 0.3568786084651947, -0.9718781113624573, 1.189882755279541, -0.9787778854370117, -0.4559497833251953, 0.7003921866416931, 0.7257806658744812, -0.9100826382637024, -0.6819656491279602, -0.3534373939037323, 2.7165825366973877, 0.4349900782108307, 0.5399531722068787, 0.03905807062983513, -0.2841989994049072, -0.028626779094338417, -0.8789215683937073, 0.12276806682348251, -0.8835535645484924, 1.5299736261367798, -1.029930591583252, 1.562171459197998, 0.09994903206825256, -1.3309434652328491, 1.0827819108963013, -1.4012936353683472, -0.6414247751235962, 0.28398263454437256, 1.0947227478027344, 1.877898097038269, -1.592647910118103, 0.8174609541893005, 1.1572891473770142, -0.18714988231658936, 0.7099916338920593, -0.2753455936908722, -0.5470176339149475, -0.5691186189651489, -1.0631608963012695, 0.5406228303909302, 0.8801099061965942, 1.442138671875, -0.49423888325691223, -0.1406109780073166, -1.396836757659912, -0.5790295004844666, 0.36157935857772827, -1.082910180091858, -0.15918616950511932, -0.054834846407175064, -1.3959641456604004, -0.2970220446586609, -0.0907793715596199, -0.5292069911956787, 1.0873247385025024, -0.38672998547554016, -0.42441511154174805, -0.5946576595306396, -1.9057774543762207, 0.9352563619613647, -0.2205866128206253, 0.23220065236091614, 1.0268723964691162, -0.08292175829410553, 0.13073062896728516, 0.25838080048561096, 0.921734094619751, 0.42961356043815613, -0.795391857624054, -1.3033106327056885, 1.4066604375839233, -1.2820457220077515, -0.03985704854130745, 1.5152711868286133, 0.48971936106681824, -0.8267080187797546, -0.4218875765800476, 0.06991781294345856, -0.6665635108947754, -0.38952961564064026, -1.7989414930343628, 1.2149373292922974, 0.5028377771377563, 0.8970596790313721, 0.33294641971588135, 1.3232686519622803, -0.8950625061988831, -0.9083831906318665, 1.1638245582580566, 0.29427194595336914, 0.422040194272995, 0.11153741925954819, -1.249462604522705, 0.19730347394943237, -0.054725080728530884, -0.20817552506923676, -0.6890735030174255, -1.2128326892852783, -0.33101576566696167, 2.0180819034576416, -0.2131459265947342, -0.5613369345664978, -0.8123265504837036, -1.2733062505722046, -0.5256058573722839, 0.3041631281375885, -0.2150854766368866, 0.9090269804000854, -1.0142384767532349, 0.38938310742378235, -0.9200571775436401, -1.0953900814056396, 0.7027201652526855, 0.09373914450407028, 0.36642953753471375, -0.7894278168678284, -0.2350536584854126, 0.473509818315506, -0.18893083930015564, 1.4801528453826904, -0.3813728988170624, 0.4098801612854004, -0.34251078963279724, -0.425598680973053, 0.04513707011938095, -0.16756099462509155, 0.33456727862358093, -0.25849366188049316, 0.4781130850315094, -1.9057692289352417, -0.43551090359687805, 0.5662127137184143, -0.08602029085159302, 1.1259868144989014, 0.6046069860458374, 0.9230302572250366, -0.8080509901046753, -0.9519257545471191, 2.002737522125244, 1.4784595966339111, -0.5299063324928284, 1.7481135129928589, -0.5869644284248352, -0.5111430883407593, -0.31888547539711, 0.6351559162139893, -1.4203301668167114, -0.6712257862091064, 0.22924764454364777, -1.626299500465393, 0.8028342127799988, -0.9863670468330383, -1.303837537765503, -0.7217611074447632, 0.5169395208358765, -0.1842944473028183, -2.038182020187378, 1.1118409633636475, 0.643517792224884, 0.40621140599250793, 0.7663386464118958, -0.4238710105419159, 1.4191279411315918, -0.8960840106010437, 1.3725099563598633, -0.13115809857845306, 0.287832111120224, 0.7299439907073975, -0.07710912078619003, -0.5370124578475952, -1.9924579858779907, 0.4274665415287018, 0.5877487063407898, 0.6195710897445679, 1.0024741888046265, 1.0224088430404663, 0.834098756313324, -0.11579549312591553, -0.45608001947402954, -1.2875033617019653, -0.2533550560474396, 1.3377233743667603, 0.5324805974960327, -1.312905192375183, -0.4063974916934967, -0.269976407289505, -0.6835232973098755, 2.3484864234924316, -0.33600834012031555, 0.40230533480644226, 0.7910738587379456, 0.3299409747123718, 1.7108874320983887, 0.38050854206085205, 0.10096053779125214, -0.16914786398410797, 0.37747788429260254, 0.841856837272644, -1.1450339555740356, 0.9444769024848938, -2.1703431606292725, -0.8922801613807678, 0.8002980351448059, -0.31437358260154724, 0.4792851507663727, -0.2225622534751892, -1.228681206703186, -0.6255533695220947, -0.10445354878902435, 1.5208261013031006, 0.565686821937561, 1.0463308095932007, 2.426495313644409, -1.3006887435913086, -1.8017481565475464, -0.4455336034297943, 0.23562441766262054, 0.9139465093612671, 1.1543446779251099, -0.9218987226486206, 0.5320351719856262, -0.8650230169296265, -1.7722737789154053, 0.4508688151836395, 1.4012142419815063, 0.03048802725970745, 0.12353114783763885, 2.263558864593506, 1.3964329957962036, 1.2932714223861694, 1.409531831741333, 0.5022245049476624, -0.6418019533157349, 0.3877141773700714, -0.3002289831638336, 0.6794949173927307, 1.2014256715774536, 0.9941251277923584, -2.101097822189331, -0.6672578454017639, -1.3018155097961426, -0.7647889852523804, -1.6785951852798462, 1.220441460609436, -1.6097652912139893, 0.5935717225074768, -1.023708701133728, -0.9237040281295776, 1.635414481163025, 1.4364699125289917, 0.0573577955365181, 0.03606748580932617, -1.3616856336593628, 1.5136160850524902, -0.5883947014808655, 0.3261972665786743, -1.5362292528152466, 1.6435106992721558, 1.044072151184082, -0.6279465556144714, -0.9761809706687927, -0.28570565581321716, 0.8838898539543152, 0.5786072015762329, 0.9754009246826172, 0.4374659061431885, 0.008251853287220001, 0.08485639095306396, -0.4190020263195038, -0.513523280620575, -0.25272679328918457, -0.8214368224143982, 0.40825775265693665, 0.8617091774940491, 0.3083023726940155, -0.9103569984436035, -0.8737043142318726, 1.1372172832489014, -0.1885654777288437, 1.5054751634597778, 0.42516785860061646, -1.883644938468933, -0.9592452049255371, -0.6842378377914429, 0.439891517162323, -0.7595685124397278, 0.8653849363327026, 1.415518045425415, -0.1631651669740677, 0.05700094252824783, -2.3131539821624756, 1.558099627494812, -0.6825527548789978, 0.3668360114097595, 0.7975472211837769, 0.988720715045929, -0.36252886056900024, 0.4143299460411072, -0.0334760881960392, -0.22866538166999817, 1.335962176322937, 1.6938263177871704, -0.9266980886459351, -0.5182758569717407, -1.7199102640151978, 0.5522729158401489, 1.0941191911697388, -0.01582791842520237, -0.5925688147544861, -0.5082014203071594, 2.156426191329956, 0.7159436345100403, -1.5991469621658325, -0.23713736236095428, -1.230419635772705, -1.2267109155654907, -0.5625470280647278, -0.18601618707180023, 0.6918821334838867, -1.6340073347091675, -0.6096293330192566, 1.7857515811920166, -1.0707470178604126, 0.4927982985973358, 1.3060719966888428, 0.40928980708122253, -2.5677084922790527, 0.002882137894630432, -0.7429865598678589, 0.44946610927581787, -0.6206575036048889, -0.6828873157501221, -0.23132239282131195, -1.6398311853408813, -1.619531512260437, -0.056224990636110306, -0.6774577498435974, 0.15984191000461578, 1.6022297143936157, -0.8008543252944946, 0.6281436681747437, 0.7562488317489624, 0.3932713270187378, -0.15303853154182434, 0.042120616883039474, 1.291718602180481, -0.9270778894424438, 0.5428522229194641, -1.1135836839675903, -0.7270276546478271, -1.5502530336380005, 0.8809636235237122, -0.747449517250061, -1.0178345441818237, 0.8826913833618164, -0.32475027441978455, 0.6870594024658203, 0.5065077543258667, 0.6705136895179749, -1.6219960451126099, -0.5346101522445679, -0.011146178469061852, 0.20787271857261658, -0.13798260688781738, -1.0211087465286255, 0.6171303987503052, -1.632682204246521, 1.2926363945007324, -0.0873798355460167, 2.1751034259796143, -0.4910428524017334, -0.11425907164812088, -1.4277758598327637, 0.7730377316474915, 0.3349669277667999, -0.5492316484451294, 0.026134731248021126, -0.05424720421433449, -0.4097428619861603, -1.093353509902954, -1.7074596881866455, -0.5553503632545471, -0.7643505930900574, 0.012371903285384178, -1.1882636547088623, 0.05553160980343819, 1.0363292694091797, 0.8832557797431946, -0.7477616667747498, -0.5520423650741577, 1.0859886407852173, 1.3225258588790894, 0.268728643655777, -1.4994779825210571, 1.1720161437988281, 0.571181058883667, 0.8883200287818909, -0.4349859952926636, -1.5879119634628296, -0.013963811099529266, -1.6462804079055786, 0.3726607859134674, -0.4720431864261627, -0.7493882179260254, 0.16285710036754608, 2.8043181896209717, 0.3066250681877136, 0.3194241225719452, -0.5462250709533691, 0.9538364410400391, 1.4376906156539917, -0.7429673671722412, 0.15333348512649536, -0.2964063584804535, 1.9743365049362183, 0.4526362717151642, 0.6274906992912292, -0.19538624584674835, 1.1016913652420044, 0.20459245145320892, 0.29672202467918396, -0.1432986855506897, 3.3788323402404785, -0.138039693236351, -0.8541596531867981, 1.4263463020324707, -1.04533052444458, -0.14082668721675873, -0.7056553959846497, -0.3336378037929535, -0.2520011067390442, 1.853455662727356, 0.3915071487426758, 0.6857022047042847, -0.841759204864502, -1.368916630744934, -0.004415035713464022, 0.8459863662719727, -1.2039018869400024, -0.24549800157546997, -0.7037305235862732, 0.11718697845935822, -0.43550437688827515, -0.3640408217906952, -0.12851670384407043, -0.3649766445159912, 0.08235947787761688, -1.1896111965179443, 1.4610121250152588, -0.9188774228096008, 1.1869239807128906, 1.5223320722579956, 0.3733212947845459, -0.6893441081047058, 0.03536004200577736, -0.4310115873813629, -1.27858304977417, 0.3659292757511139, 0.3648194372653961, 0.261134535074234, 0.6060419678688049, 0.7187613844871521, -0.3030060827732086, -1.0506511926651, -1.0167425870895386, -0.004539186134934425, 0.5297364592552185, -0.9453564286231995, -0.12921081483364105, -1.496098518371582, 1.5520415306091309, 0.4977743327617645, -0.39888060092926025, -0.9335992932319641, -0.6963770389556885, 0.9960245490074158, -0.05879297852516174, -0.2695154845714569, 0.08943842351436615, -0.46519935131073, 1.3463478088378906, 0.33650076389312744, 0.2086559385061264, -1.7411930561065674, 0.2457948625087738, 0.7190005779266357, -0.8356059193611145, -1.3490277528762817, 0.8583317399024963, -1.0688084363937378, 0.4380126893520355, -0.07025446742773056, 0.2858797311782837, 0.378341406583786, 0.23989132046699524, 2.761366367340088, -0.613158643245697, -0.12922817468643188, -1.4968518018722534, -0.14237742125988007, -0.7146204710006714, -1.1133463382720947, 0.342012882232666, -0.36613425612449646, -0.8639280200004578, -0.25765717029571533, -0.7386599183082581, 3.7896647453308105, 0.006414669565856457, -0.15674200654029846, -0.5338376760482788, -0.3899036645889282, -0.20414841175079346, 1.2914323806762695, 0.5403353571891785, 1.5300991535186768, -0.6250133514404297, -0.6723766326904297, 0.035759229212999344, -0.2255128026008606, 1.41484797000885, -0.7403748035430908, -1.3195165395736694, 1.3821238279342651, -1.5408324003219604, -0.7558021545410156, -1.1321766376495361, 2.2704038619995117, -0.9091241359710693, -0.7860302329063416, -1.1796241998672485, -1.4165340662002563, 0.16531695425510406, -1.662218451499939, 1.2151522636413574, 1.4271806478500366, 0.3633492588996887, -0.38642069697380066, 0.4413364827632904, 1.5382883548736572, 0.6151176691055298, -1.9672060012817383, -0.4985232651233673, -0.20783331990242004, -1.2945302724838257, -1.526906132698059, 0.9097630381584167, -0.28561997413635254, -0.9031352996826172, 2.028378486633301, 0.37351125478744507, 1.5673694610595703, -0.05538899451494217, 0.5643608570098877, 1.6041301488876343, 0.2983156144618988, 1.3623299598693848, -0.3473755419254303, 0.09765167534351349, 0.9204035997390747, -1.2124574184417725, 1.2745634317398071, 1.8774970769882202, -1.5626789331436157, 1.4551432132720947, 0.07881861925125122, -0.9654240608215332, 0.1722121387720108, 0.3428777754306793, -1.2472211122512817, 1.5341424942016602, -0.5174989700317383, 0.2851378917694092, -0.4923519194126129, 0.9838768243789673, -0.9774981141090393, -0.17224812507629395, 0.29411494731903076, 1.126685619354248, -1.8794809579849243, -0.5453296303749084, -0.2130049616098404, -1.2435328960418701, 0.4123644232749939, -1.4059429168701172, 0.9917994141578674, 0.2694730758666992, -1.401496410369873, 1.539335012435913, -0.13073650002479553, -2.202064275741577, -0.07817631959915161, -0.6043229699134827, 1.236193299293518, 0.3611186742782593, 0.01709211803972721, 1.2510923147201538, -1.3720381259918213, -1.5828828811645508, 1.1876195669174194, -0.8431320190429688, 1.0477076768875122, -0.9470675587654114, -0.5955442190170288, -1.827662467956543, -0.2812991738319397, -2.0204339027404785, -2.050692558288574, -1.1438966989517212, 1.3686796426773071, 2.237999439239502, -1.1047320365905762, 0.08918315172195435, 0.7217774987220764, -1.1502513885498047, 1.169848918914795, 0.056230586022138596, 1.5710632801055908, -0.7956089973449707, 1.2163748741149902, -0.3525477945804596, -0.5509551167488098, -0.8779939413070679, 0.9424623250961304, -0.8603179454803467, -0.9066709876060486, -0.6939782500267029, -0.7086432576179504, -0.28906041383743286, 1.5080945491790771, -0.5032057166099548, 0.6379900574684143, -1.1994454860687256, -1.0359110832214355, -0.7784467935562134, 0.7734938859939575, -0.7525455355644226, 0.0485377162694931, 0.8342557549476624, -1.212874412536621, -0.41317465901374817, 0.05269157886505127, -0.8433148860931396, -0.5257390141487122, 0.8889737725257874, -0.6827538013458252, 1.0515732765197754, 0.19684767723083496, 2.3473362922668457, 0.1639600694179535, 0.7914035320281982, 1.6893197298049927, 0.8155768513679504, -0.49501216411590576, 0.7995867133140564, 1.828896403312683, -0.5805046558380127, -1.333829402923584, 0.7254712581634521, 0.26466625928878784, -1.2568583488464355, 1.1369701623916626, -0.7501211166381836, 1.1866244077682495, -0.06608438491821289, 0.2331201136112213, 0.05890514329075813, 1.0751553773880005, 0.5545533299446106, 0.09094267338514328, -0.47417619824409485, -0.38965827226638794, -0.37255629897117615, -1.1290185451507568, 0.7949648499488831, 0.6427140235900879, -0.4477063715457916, -0.678856611251831, -0.536812424659729, 0.23373113572597504, -0.8730323314666748, -0.6498338580131531, 1.609916090965271, -0.20328272879123688, -0.45900389552116394, -0.00044868633267469704, -1.3786643743515015, -2.60843563079834, -0.39409980177879333, 1.2532258033752441, 0.29640015959739685, -1.1588398218154907, 1.0525870323181152, 0.6703192591667175, -0.6043822765350342, 0.9906357526779175, -0.9051440954208374, 0.03756579011678696, 0.7406315803527832, -0.8868589401245117, -0.5626034140586853, -0.4676312804222107, -1.1429007053375244, 1.0062634944915771, 1.153695821762085, 1.3335078954696655, 0.03090113215148449, 0.3914664685726166, 1.0794918537139893, 0.40231478214263916, -1.3645235300064087, 1.587083101272583, 0.01494444627314806, 0.08928484469652176, -1.1201940774917603, -1.408516526222229, 0.8699895143508911, 0.3929812014102936, -0.619744062423706, 1.2234773635864258, 0.04788374528288841, 1.3630647659301758, -0.8720300793647766, -0.2080804854631424, 0.6463603377342224, 0.3709700107574463, 1.147415280342102, 0.7184569239616394, 0.864311158657074, 3.1206674575805664, 0.001137211686000228, 0.2719365954399109, 0.8551873564720154, 0.0365665927529335, 0.4910004436969757, -1.0009528398513794, -0.5210745930671692, -2.0966012477874756, -0.06492003053426743, -0.42001405358314514, -1.4222322702407837, 0.29440420866012573, -0.6356819272041321, -0.9350640177726746, 0.1931387484073639, 0.2532272934913635, 1.3458874225616455, -0.8016520142555237, 1.6435248851776123, 1.2268257141113281, 2.1439619064331055, 0.9356425404548645, -0.485102117061615, 0.2550988495349884, -1.294135570526123, -0.7936386466026306, 1.0805678367614746, -0.22049470245838165, -1.168766736984253, -0.07871678471565247, 1.690639853477478, -0.8279121518135071, 0.5509355664253235, 0.11507365852594376, -1.3687844276428223, -0.41619032621383667, 0.0036904450971633196, -0.08381595462560654, 0.5895370244979858, 1.0190980434417725, 0.5489601492881775, -0.9526585936546326, -1.0597642660140991, 0.7608826160430908, 2.8184762001037598, 0.06272342801094055, 0.7187570333480835, -0.38449758291244507, 0.8528684973716736, 0.36239296197891235, -0.354300320148468, 0.30434539914131165, -1.6460237503051758, -0.02004741132259369, -1.2907679080963135, -0.7333559393882751, 0.1885925531387329, 0.9688736200332642, 0.03349587321281433, -1.2192928791046143, 0.08447976410388947, 0.8690597414970398, 0.6056842803955078, 0.3263620436191559, -1.026990294456482, -1.5088927745819092, -0.06874259561300278, 0.4398438036441803, -0.8320719003677368, -0.8268362283706665, -1.1231462955474854, 0.826743483543396, 0.4601963460445404, -0.4082771837711334, -1.4643806219100952, 0.7510669827461243, 1.0652364492416382, 0.38043123483657837, 0.6688551902770996, 1.7629033327102661, -1.3440508842468262, 0.32115423679351807, -1.3455607891082764, 1.4480844736099243, 0.6485865712165833, 0.42163437604904175, 0.40161561965942383, 0.10158741474151611, 0.5868804454803467, -0.6394264698028564, 1.8258254528045654, 1.6191859245300293, 0.6932895183563232, -0.19924764335155487, 0.10433237254619598, 0.27927106618881226, 1.3787193298339844, 0.48767417669296265, 0.4260982275009155, -0.03092687390744686, -0.41251125931739807, -1.1010164022445679, -0.008747025392949581, -0.7548929452896118, 0.7741004824638367, -0.6850314140319824, 1.0071947574615479, -0.6903495192527771, -1.6768221855163574, -0.5141274333000183, 0.8297008275985718, 1.3120172023773193, 1.6964912414550781, 1.812262773513794, 0.2560517191886902, 1.4450252056121826, -0.205598846077919, -0.40735548734664917, 0.30838868021965027, -0.30331069231033325, 1.1852946281433105, -1.0660104751586914, -0.532249927520752, -0.6761131286621094, 1.1091545820236206, -1.0009214878082275, 0.5462714433670044, -1.1789976358413696, -0.3073200583457947, -1.6090251207351685, 1.9259618520736694, 1.9753522872924805, -0.18666431307792664, 1.0296396017074585, 0.2728244364261627, -0.9771549701690674, -1.4597351551055908, -1.031473159790039, 0.6473228931427002, 3.5949549674987793, -0.466398149728775, -0.7578229308128357, -0.5633202195167542, 0.03256768733263016, 1.339368224143982, 1.0760602951049805, -0.4401107430458069, 1.4874728918075562, -0.2688325345516205, -2.511807680130005, -0.07871613651514053, -0.8438553214073181, 1.6232600212097168, 1.1029388904571533, -0.8654105067253113, 0.20946086943149567, -0.08019759505987167, 0.3628201484680176, -0.39468738436698914, 0.6973878741264343, 1.684952974319458, -0.5126886963844299, -0.4374299943447113, -1.4348454475402832, 1.4333430528640747, 0.7314529418945312, -0.13726447522640228, -1.143704891204834, 0.7319345474243164, -1.4297220706939697, -0.12453897297382355, -2.1281654834747314, 0.9138179421424866, -0.42807477712631226, -1.5749561786651611, 0.8742086887359619, 0.0874996930360794, -1.7612565755844116, -1.0199707746505737, 1.1542335748672485, 1.673179030418396, -0.27960914373397827, 0.8370154500007629, -0.9433608651161194, 2.003089427947998, -0.4430938959121704, 0.7982232570648193, 0.8642737865447998, -0.01934603601694107, 1.7352628707885742, 0.024883229285478592, -0.34818705916404724, -2.0345311164855957, 1.2531074285507202, 0.4278109669685364, -1.290520429611206, 2.193195104598999, -0.4075651168823242, -0.7331520318984985, 0.7166181802749634, -0.23495854437351227, -0.23570260405540466, 1.714219570159912, -0.09494689106941223, -0.39579880237579346, 0.06653641164302826, 0.5504438281059265, -0.7662435173988342, 0.3164953589439392, 0.5148665308952332, 0.769012451171875, -0.8704279661178589, 1.1727657318115234, -1.2098362445831299, 0.8531339168548584, 1.1571744680404663, -1.2814598083496094, 0.3161107003688812, -0.195419579744339, -0.4592866599559784, 0.3141157925128937, 0.2712344229221344, -0.7018914222717285, 1.178884506225586, 0.6836831569671631, -0.4374762177467346, -0.19834518432617188, -0.47543278336524963, 1.7646257877349854, -0.5963066220283508, 0.6879705190658569, 0.11676561832427979, 0.030241571366786957, 1.3170863389968872, -2.1795108318328857, 1.260977864265442, 0.0690164640545845, -0.7068982720375061, -0.41520532965660095, 0.3384298086166382, -0.6637035012245178, 1.1923046112060547, 0.238239586353302, 0.6391605138778687, 1.3518730401992798, -0.8414399027824402, 0.47948095202445984, -0.24543115496635437, -0.4848749339580536, 0.5484453439712524, -0.07770861685276031, -0.6104666590690613, -1.7359604835510254, -1.144650936126709, 0.4105794429779053, 1.1426209211349487, -0.0021119124721735716, -0.1525821089744568, -0.7947142720222473, -0.2976139485836029, -0.4464119076728821, 0.3454841673374176, 0.32228320837020874, -1.7602224349975586, -0.23474332690238953, -0.6804088354110718, 0.5439611077308655, 0.871473491191864, -0.8197897672653198, -0.05054780840873718, 1.2163078784942627, -1.7598425149917603, -0.8283255696296692, -0.899149477481842, -0.8752253651618958, 0.27631714940071106, 0.9146451950073242, 0.9919556975364685, 0.531582772731781, -0.3423389494419098, -0.3150346577167511, -1.3749362230300903, -0.8547372817993164, 0.7507753968238831, -0.684775173664093, 0.6848353147506714, 0.7325833439826965, -0.5690020322799683, 0.27867352962493896, -0.6321259140968323, -0.3291690945625305, -1.855417013168335, -0.11390133202075958, -0.3952251374721527, -0.41393333673477173, 0.22544682025909424, 1.0383745431900024, 2.060299873352051, -0.10013169795274734, -0.6404725909233093, -0.8274314999580383, -0.8418830633163452, -0.7898131012916565, 1.3325711488723755, -0.3572155237197876, 0.11258821189403534, 1.3966301679611206, 0.24579842388629913, 0.5292003750801086, 1.1162165403366089, 0.09924047440290451, -0.9532126188278198, -0.487571120262146, -0.4477355182170868, 0.20424684882164001, 0.4193870425224304, -1.1569267511367798, -0.7137082815170288, 0.7411359548568726, 1.1977585554122925, 0.6140614748001099, -0.23386454582214355, -0.4226476550102234, -1.174821376800537, 1.3517045974731445, 0.6529694199562073, 0.1229826956987381, 0.5325093865394592, 0.8215263485908508, 0.48294615745544434, -0.34338515996932983, 1.8947854042053223, -0.41033777594566345, -1.1162890195846558, 0.48198363184928894, -1.6330549716949463, 1.4077309370040894, 0.5143483281135559, 0.21833071112632751, -0.7738755345344543, -0.30960318446159363, -1.333350419998169, 0.04133250191807747, 0.49442410469055176, 1.743552327156067, -0.28219109773635864, 0.9873917102813721, 0.4757237434387207, 1.3226734399795532, 1.3720487356185913, 0.7868326306343079, 0.6143631339073181, -1.2390072345733643, -0.24890238046646118, 0.3093222975730896, 0.39621302485466003, 0.848281979560852, 1.2513540983200073, -1.0417637825012207, -0.29872217774391174, -0.9954853653907776, -0.7849984169006348, -0.07371051609516144, -0.22186318039894104, 1.5120279788970947, -0.663424551486969, -0.894040584564209, -3.1509997844696045, 0.21147477626800537, 0.16341358423233032, -1.0311461687088013, 0.8539716601371765, -1.6475154161453247, 0.9323357343673706, -0.604084312915802, -0.8442041873931885, 0.46292540431022644, 0.38886329531669617, -0.019496800377964973, 1.3152414560317993, -0.48942556977272034, 0.6047492623329163, 0.291515588760376, 0.08680270612239838, -0.2822912931442261, -0.753036379814148, -0.14577405154705048, -0.9241253733634949, 0.6787844896316528, -2.681910753250122, 0.35106292366981506, -1.1707804203033447, 0.7627179026603699, -1.653099536895752, 0.9558864235877991, -0.41736581921577454, 0.6486005783081055, -0.8679965734481812, -0.6971336007118225, 0.03331025689840317, 1.1026732921600342, -1.0490329265594482, 1.335741639137268, 2.8094236850738525, 0.5818295478820801, -0.12392158061265945, 0.9432442784309387, 0.012104975990951061, 0.4270925521850586, -0.8068810105323792, -0.10547082871198654, -0.3488943874835968, 0.12298262864351273, 2.373311758041382, -0.40207958221435547, -0.8782449960708618, 1.4984172582626343, -1.6937226057052612, -1.1559382677078247, -0.4166017770767212, -0.37129831314086914, 0.4281142055988312, 1.141932725906372, -0.16144369542598724, -0.02317827381193638, 0.6367643475532532, -0.32682594656944275, -1.6992981433868408, -0.9594389796257019, 0.00810338743031025, -0.038201674818992615, 0.8393438458442688, 0.8742787837982178, -0.004039326682686806, 1.1155365705490112, -0.34722256660461426, -1.4411580562591553, -1.350392460823059, -0.9196447730064392, 0.966788649559021, -1.4679443836212158, -0.6745753288269043, -2.2166502475738525, -0.32760652899742126, -0.45139196515083313, 0.9399033784866333, -0.6668039560317993, -0.8011031746864319, -0.4015757143497467, -1.1748536825180054, -0.3516162931919098, -0.06768302619457245, -1.3660380840301514, 0.9385077953338623, 1.3568257093429565, -0.9064638018608093, 1.0133970975875854, 0.5680108070373535, 0.7832220196723938, -0.17568090558052063, -0.6050534844398499, 2.1212825775146484, 0.14423337578773499, 0.8208902478218079, -1.409323811531067, -1.3469585180282593, -0.04114518687129021, 2.6903209686279297, -1.7178627252578735, 0.6197727918624878, 1.6108530759811401, -1.1826934814453125, 0.7632675170898438, 0.9271316528320312, 0.5568808913230896, 0.13333861529827118, -0.23630447685718536, -0.02302195318043232, 0.8127471208572388, 0.8959407210350037, -0.9505214095115662, 0.32874932885169983, -0.14511916041374207, -1.0860127210617065, -0.13685764372348785, -0.6248524188995361, -0.0903487429022789, 1.8571027517318726, 0.5704109072685242, 1.291939377784729, -0.9816812872886658, 1.922843098640442, -0.770548403263092, -0.5461075305938721, 0.032646920531988144, 0.6789682507514954, 0.4147307872772217, 0.6110007166862488, -1.3683555126190186, -0.020102694630622864, -0.3682595193386078, 1.3148763179779053, -0.2973463833332062, 0.7830062508583069, -0.6259347796440125, -1.6273455619812012, -0.5000970363616943, -1.6378637552261353, -0.24627423286437988, 1.0398226976394653, 1.554581642150879, -0.6296624541282654, -0.16529667377471924, 0.28408411145210266, 1.3234877586364746, 1.6814581155776978, -0.07477834075689316, 0.06770967692136765, 1.0135048627853394, -1.5461112260818481, -0.1578822135925293, -0.11603527516126633, 1.0468753576278687, 0.33318936824798584, -0.389386385679245, -0.4152100086212158, 1.7233611345291138, -2.2830116748809814, 0.6759769916534424, -0.6598708033561707, 1.4331011772155762, 0.6981292963027954, 0.7152929902076721, -0.4038456678390503, 0.30179905891418457, 0.7316473722457886, -1.260722041130066, -0.2431911677122116, -0.7416719198226929, 0.3546009361743927, 1.3714677095413208, -1.3209035396575928, 2.259556531906128, -0.9249814748764038, -2.04215407371521, 0.2528333067893982, 0.4411042332649231, 0.9432043433189392, 0.39393818378448486, 0.6441637873649597, -0.7184860706329346, -0.16871118545532227, 0.786643922328949, 0.617855966091156, 0.6837126612663269, -1.6061546802520752, -1.301137924194336, 0.3616291880607605, -0.2444252222776413, 2.526780843734741, 0.4754265248775482, -1.510581374168396, -0.5516060590744019, -0.36429789662361145, 0.014659176580607891, 0.5957601070404053, 0.4641515910625458, 0.4409126341342926, -0.5947904586791992, -0.16961883008480072, 0.04781734570860863, 0.7492239475250244, -1.3045473098754883, 1.2664093971252441, -0.9416089653968811, 0.1111319288611412, 0.5052143931388855, -0.37027621269226074, -1.5002822875976562, -0.44894739985466003, 0.2893090844154358, 0.13002179563045502, -0.3407253921031952, -0.5152269601821899, 0.7455801367759705, 0.03928125277161598, 0.7135137915611267, -0.8598344326019287, -0.7227223515510559, -0.7105805277824402, -0.024995049461722374, -0.1470394879579544, 0.22549192607402802, 0.113780677318573, -1.4626400470733643, 0.9066266417503357, 0.01875951699912548, -1.1061779260635376, -2.225273609161377, 1.4325408935546875, 0.7634931802749634, -0.48170650005340576, -0.43032947182655334, 0.07354909181594849, 0.9086906313896179, 2.1446280479431152, -0.6734294891357422, -0.9355512857437134, -0.07557535916566849, -0.7621660232543945, -0.5058082938194275, -2.025322914123535, 0.46572408080101013, -1.6446863412857056, -0.7886863946914673, -0.0032609535846859217, 1.4768152236938477, -2.000020742416382, 0.672132134437561, -0.14724436402320862, 0.6658431887626648, 2.3774046897888184, 0.1355152577161789, -0.08584103733301163, 0.09703440964221954, -0.6572864055633545, -0.7540358901023865, 0.9997968077659607, -0.6301112174987793, 1.1191726922988892, -0.6895870566368103, 1.4148122072219849, -0.5301111936569214, -1.8088023662567139, 0.11532475799322128, -0.1969706416130066, -1.1862728595733643, 1.0739120244979858, -1.4625263214111328, 1.6880170106887817, -0.42778193950653076, 0.8251497149467468, -0.5762145519256592, 0.1576455533504486, 0.8271750807762146, 1.4623411893844604, -0.9089822173118591, 0.4049718976020813, -1.0158241987228394, 0.37470516562461853, 0.843425989151001, 0.6228086352348328, -1.362217664718628, 0.20922192931175232, 0.5227031111717224, 0.12531901895999908, 1.3472031354904175, 1.21578848361969, -1.2540555000305176, 0.6897420883178711, 0.7539142370223999, -0.771745502948761, -1.779715657234192, -1.4452558755874634, 0.6345988512039185, -0.6639096736907959, 2.2055249214172363, -0.9624272584915161, 0.6141199469566345, 0.6132838726043701, 0.44235360622406006, 0.47633445262908936, 0.860651433467865, -0.8254655003547668, 0.0126802958548069, -1.6908235549926758, 0.42997244000434875, 1.1003221273422241, -0.30943018198013306, -1.7584961652755737, 0.23029237985610962, -0.2993411719799042, 0.693252682685852, 0.304094135761261, -0.9795406460762024, 0.8548213243484497, -0.26539817452430725, -1.3111586570739746, 0.6515570878982544, -0.6535537838935852, 1.3899788856506348, 1.168984055519104, 1.1403709650039673, 1.4435973167419434, 0.539043128490448, -0.9498780965805054, 0.38499191403388977, 1.0895717144012451, 0.8919786810874939, 0.5974328517913818, 1.0686134099960327, -0.9956422448158264, -0.7392775416374207, 0.23446844518184662, -0.77448970079422, 1.742603063583374, -0.06682552397251129, -1.247488021850586, -0.3968566954135895, 0.6712383031845093, 0.023686323314905167, -1.1911760568618774, 0.10619322955608368, -0.46547022461891174, -1.169385313987732, -0.0686667338013649, -1.3271623849868774, -0.933266282081604, -1.6494945287704468, -0.25052589178085327, -0.3032767176628113, 0.8951970934867859, -1.010321021080017, 0.497372031211853, -0.9368414878845215, 0.7254644632339478, 0.366363525390625, 0.4684714078903198, -0.2601604461669922, 0.3502576947212219, -2.1380746364593506, -1.3322995901107788, 1.344861626625061, -0.6103225946426392, 0.0020254431292414665, 1.181409478187561, 1.447503924369812, 0.25313636660575867, -1.8674290180206299, 0.7656710743904114, -0.11673435568809509, -0.3800036907196045, 1.8374203443527222, 0.4526686668395996, 0.6459456086158752, -2.010944128036499, -0.5281452536582947, 0.8578203320503235, 0.2433617264032364, -0.40130850672721863, 1.2192282676696777, 0.17502781748771667, -0.9881114959716797, -0.04834659770131111, -0.6157063245773315, 1.0440242290496826, 0.6424925327301025, -0.38634341955184937, -0.23975403606891632, -2.0873842239379883, -1.1118642091751099, -1.6898179054260254, -0.4597649872303009, 0.33804258704185486, -2.1008410453796387, -1.1589080095291138, 0.9721843004226685, -1.8424568176269531, 2.318943738937378, -0.5668392181396484, 0.5924491286277771, 0.8553048968315125, 0.548673689365387, 1.7183029651641846, -0.9630798101425171, -0.1467503160238266, -0.7745380401611328, 0.6005368232727051, -0.46381404995918274, 1.3144433498382568, 0.6167640089988708, 0.7491637468338013, 0.8577916026115417, -0.41599270701408386, -0.7180396318435669, 0.17746834456920624, -0.5635838508605957, -1.0281175374984741, -0.10023507475852966, -0.017701277509331703, -1.7126753330230713, 1.2701910734176636, 0.11178485304117203, -0.8810380697250366, -0.336390882730484, 0.5305615663528442, 1.1256481409072876, -0.7773864269256592, -0.7090023756027222, -0.2576626241207123, 0.037717003375291824, 1.1999104022979736, 1.1262824535369873, -0.6019032597541809, -0.04025834798812866, -0.9869098663330078, -1.4583182334899902, 0.014036286622285843, -0.22329849004745483, -0.5843586921691895, -0.8887853622436523, 0.18473899364471436, 0.21712103486061096, 1.6215078830718994, -0.6755454540252686, -0.6012771129608154, -0.41007643938064575, 4.3176188468933105, 0.8811036348342896, -1.8522803783416748, 0.23037676513195038, -0.3997696340084076, -2.034473419189453, -0.9305092096328735, 0.09091866761445999, 1.9976834058761597, 0.39077454805374146, 1.0734220743179321, -1.8187286853790283, -0.26888978481292725, -0.5704087018966675, 1.5563158988952637, -0.4228500425815582, 0.26306894421577454, -0.6080401539802551, -0.11197277903556824, -1.70982027053833, 1.102053165435791, -0.8649424314498901, 0.034683939069509506, -0.952636182308197, 0.29797545075416565, 0.008841480128467083, 0.38414955139160156, -0.9387180209159851, 0.20762355625629425, 2.3342509269714355, 0.4469888508319855, 0.6128399968147278, -0.30611056089401245, -0.8507452607154846, 0.7586404085159302, -0.654950737953186, -1.1501998901367188, -1.32452392578125, -1.6608500480651855, -0.9140428900718689, -0.8723201155662537, 1.3134987354278564, 0.4700702428817749, 0.5082067251205444, 0.6553757190704346, -0.7530469298362732, -0.6220251321792603, 0.03174066171050072, -0.8677511215209961, 0.5308386087417603, -0.4150020182132721, -0.24551373720169067, -1.3524552583694458, -0.17061682045459747, -2.178892135620117, 1.408915638923645, -0.32078152894973755, -0.20066674053668976, 0.013480216264724731, 0.5423849821090698, -2.4061169624328613, -1.3796257972717285, 1.0393245220184326, 0.9543314576148987, -0.4059500992298126, -0.5317410230636597, -1.0990910530090332, 0.05114596337080002, -0.6776437759399414, 0.451871395111084, 0.7485758662223816, 0.06347616761922836, 0.0895310640335083, -2.942033052444458, -0.8733150959014893, -0.5090333819389343, 0.1432446837425232, 0.4143916368484497, -0.20450222492218018, 0.45750781893730164, 0.05031505227088928, -0.7946294546127319, 0.5427929162979126, -1.0436229705810547, -0.6055218577384949, -0.024660365656018257, 1.5126783847808838, 1.0502609014511108, -0.4007876217365265, 0.13384892046451569, 0.701612651348114, -1.8873438835144043, -0.2251710146665573, -1.1818772554397583, -0.589058518409729, 2.0858020782470703, -1.1231005191802979, -1.139402985572815, 0.9596447348594666, 0.7320978045463562, 0.7579843401908875, 0.26677483320236206, 1.1665242910385132, 0.08126173913478851, -1.1369777917861938, -1.0045990943908691, 0.45504769682884216, 0.1889941394329071, -0.11760120093822479, -0.31660300493240356, -0.33271822333335876, -0.5077804327011108, -0.5376747846603394, -1.2276639938354492, 0.7053084373474121, -0.3689940273761749, 0.8411840200424194, -0.8808578848838806, -1.1140801906585693, 0.45686590671539307, -0.19815459847450256, 0.45178642868995667, -0.6162403225898743, -0.009755074046552181, 0.541056215763092, -0.11375758796930313, 1.626635193824768, 0.2526305615901947, 0.37570810317993164, 0.12343091517686844, -1.7779451608657837, 0.4256916046142578, -0.9478769898414612, -0.5269264578819275, -0.2630175054073334, -0.9614577293395996, 1.6846457719802856, 1.7362456321716309, 0.03076254203915596, -0.28804439306259155, -0.6584044098854065, -0.8560634255409241, 1.3760217428207397, 0.734710693359375, 0.919152557849884, 0.018755581229925156, -2.3845958709716797, 1.0552808046340942, -0.3155837953090668, 0.24634912610054016, -1.222182273864746, 0.08835319429636002, -0.39377889037132263, 1.2896084785461426, 0.43637093901634216, -0.34253379702568054, 0.11346820741891861, 0.34405606985092163, -0.19823379814624786, -0.2213340848684311, 0.08537189662456512, 0.49097222089767456, -2.2153568267822266, 0.49169620871543884, -0.7917382717132568, 0.29369693994522095, 0.744208037853241, -0.7000375390052795, 0.9017828106880188, -0.03780949115753174, 0.06997943669557571, -0.23149748146533966, -0.2799910008907318, -1.1319940090179443, 0.2628500759601593, -1.2082328796386719, -1.2107088565826416, 1.2496848106384277, 0.9451954960823059, 2.3724093437194824, -1.1201891899108887, -0.7774192690849304, 1.2774858474731445, -1.680464267730713, 1.4500541687011719, 1.1040395498275757, 0.46769335865974426, -0.07286491990089417, 0.050376810133457184, -0.6395597457885742, 0.295460045337677, 0.17324812710285187, 0.27731069922447205, -0.03266622871160507, 0.7591243386268616, 0.4218601584434509, -1.0482107400894165, -0.9913359880447388, -0.296196311712265, -0.6428263187408447, -0.45707938075065613, 0.9178079962730408, 2.080876350402832, 1.0895395278930664, -1.9488067626953125, 0.14559538662433624, 0.26861947774887085, 0.8449367880821228, 0.7754440307617188, 0.35056012868881226, -0.04179834946990013, 1.5596439838409424, 1.4925655126571655, 1.130256175994873, -0.08516617119312286, 1.2302500009536743, -0.8708228468894958, 0.3245508074760437, 1.1705851554870605, 0.34974291920661926, -1.006242036819458, -0.9465977549552917, -1.3830938339233398, -0.7794742584228516, 0.0904177874326706, 1.527230978012085, 0.18806935846805573, 0.14740724861621857, 1.8804678916931152, 0.43953096866607666, -0.5524875521659851, -0.04557519778609276, -0.3931101858615875, -1.1174044609069824, 0.34539151191711426, -1.048828363418579, 0.39657822251319885, 1.0584913492202759, -0.4385138750076294, -0.8004763722419739, -0.047264788299798965, -0.55733323097229, 0.059241242706775665, 0.6312457323074341, 0.5183071494102478, 0.19642391800880432, -1.2430214881896973, 0.4918675422668457, -1.5241944789886475, 0.03124268166720867, 1.365648865699768, -0.7337307333946228, -1.155158281326294, -2.4743800163269043, 1.8121834993362427, 0.49789583683013916, -1.8566408157348633, 1.7119625806808472, 0.5336116552352905, -0.758029580116272, 0.7067430019378662, -1.0528618097305298, -1.464113712310791, -0.4388803541660309, 0.05223781242966652, 0.9106728434562683, -0.16295126080513, 0.16040560603141785, 0.04043335095047951, -0.09465735405683517, 1.0954338312149048, -0.08386148512363434, -0.8703456521034241, 0.2684330642223358, 0.10884921997785568, 0.3533822298049927, -0.1986943930387497, -0.07450634241104126, 0.6882397532463074, 0.9913434982299805, -1.007788896560669, -1.5720502138137817, -0.6899292469024658, -0.45346856117248535, -1.1438698768615723, -0.9240174293518066, 0.25320297479629517, -0.21761803328990936, -0.06764394044876099, -1.7556968927383423, -1.006245732307434, -0.9182400107383728, 0.0062758647836744785, 1.3870760202407837, -2.0808844566345215, 0.8380023837089539, -1.4158321619033813, -0.23911917209625244, 0.4120502471923828, -1.571972131729126, -0.9493461847305298, 2.144397735595703, 0.39865899085998535, 0.5683072805404663, -0.787668764591217, -0.4713278114795685, -0.34249061346054077, -0.5837249755859375, -2.1691296100616455, 1.8188517093658447, 0.577427089214325, -0.06506083160638809, -2.2581872940063477, 0.5731312036514282, -0.0863056555390358, -0.22974757850170135, 0.599014163017273, -0.2601131200790405, -0.35103631019592285, 0.6442437171936035, 0.2617078721523285, 1.3919322490692139, -0.8953238129615784, -1.0627644062042236, 0.5730509161949158, -0.7540618181228638, 0.5820738077163696, 1.800520658493042, 0.9065081477165222, -0.7365075349807739, -0.5156030058860779, -0.2893209755420685, -0.49151402711868286, -0.1049751341342926, 0.1862112432718277, -0.07016107439994812, 0.08662951737642288, -1.1716465950012207, 2.1500203609466553, -0.21875175833702087, -1.6489304304122925, -1.1452815532684326, -0.6897614598274231, -0.8554872870445251, -0.7606265544891357, -0.030542809516191483, 0.23733051121234894, 1.5629457235336304, -1.4929178953170776, -1.5241683721542358, -0.01934373565018177, -1.097913146018982, -0.6796526908874512, -1.6757540702819824, -0.22355620563030243, 2.4116880893707275, 0.658066987991333, -0.6361907124519348, 1.9037448167800903, -2.1248695850372314, -1.505888819694519, -0.33924633264541626, -0.7419872283935547, 0.09166046231985092, -0.04697352275252342, -1.538138508796692, -1.24580979347229, -1.4496251344680786, -0.9433232545852661, 0.6002871990203857, -0.9918553233146667, -0.5697107911109924, 0.44680535793304443, -0.985660970211029, 1.8253098726272583, -0.29080072045326233, -0.581657350063324, 0.7843846678733826, -0.19351309537887573, 1.276002049446106, 1.6990936994552612, 0.42475995421409607, 1.512441635131836, 1.4787112474441528, -0.42269662022590637, 2.308346748352051, -0.25219205021858215, 0.8841778635978699, -0.3282042443752289, -0.24407783150672913, -0.02025046944618225, -0.06402692943811417, 0.09213783591985703, 0.9958000779151917, 0.5096185207366943, -1.9057645797729492, 0.07929737120866776, -0.3379098176956177, 1.0640813112258911, -0.38283857703208923, 0.19509129226207733, -0.4287341833114624, -0.12363462150096893, 1.4501224756240845, 0.2986154556274414, 1.0320696830749512, 0.1626143604516983, -0.9449169039726257, 0.3890089988708496, -1.9258830547332764, 0.856637716293335, -0.46362656354904175, -2.3254239559173584, 0.35634633898735046, 1.8963720798492432, -0.15547339618206024, -0.10783270001411438, 1.7145168781280518, 0.03517425060272217, -0.17356298863887787, -0.1625448763370514, 0.6192856431007385, -0.07568210363388062, -0.891448974609375, -0.47752606868743896, -0.39728862047195435, 0.9974278807640076, -0.10394632071256638, -0.3978472948074341, 1.481375813484192, 1.0682542324066162, 0.5654791593551636, 0.3094862103462219, -1.0970298051834106, 1.312962293624878, 0.10041960328817368, 0.13325753808021545, -0.8228005170822144, -0.3338324725627899, 1.4389688968658447, 0.027228524908423424, 0.33258122205734253, -1.3279781341552734, 1.8225815296173096, 0.48354220390319824, 1.6496448516845703, -1.9701186418533325, -0.3515259921550751, -1.629658818244934, 2.2647488117218018, 1.7833832502365112, -2.1123218536376953, -0.18971161544322968, -1.1520851850509644, -0.1787775456905365, 0.48321136832237244, -0.39415377378463745, 1.119091510772705, -1.5828373432159424, -0.09900327771902084, 0.6356881856918335, -0.8855932950973511, 0.48356473445892334, 0.28086861968040466, -0.6369174122810364, -1.932342529296875, -0.2802661061286926, -0.12959399819374084, -0.8502345085144043, 1.0276808738708496, 0.006990349851548672, 0.4863191246986389, -1.027069330215454, 0.14094947278499603, 0.6927112936973572, -0.7391475439071655, -0.4597601890563965, 0.8210856318473816, 1.6463985443115234, 1.2536182403564453, 0.10553839057683945, -0.5952879786491394, 0.08386775106191635, -0.8887174725532532, 1.1998660564422607, -1.1609777212142944, 0.16656160354614258, -1.2164722681045532, 0.372478723526001, 0.022901451215147972, 0.3719198405742645, -0.7877941727638245, 0.2503448724746704, 1.5126231908798218, -0.2673235535621643, -1.5809060335159302, 0.11441277712583542, 1.6333556175231934, 1.4689465761184692, -1.0922714471817017, -0.32982856035232544, -0.6501445770263672, 0.6656296849250793, 0.34216445684432983, -0.4511238932609558, -0.16386544704437256, 0.2991338074207306, -0.7362145781517029, -0.26730507612228394, 2.2563390731811523, 0.19286799430847168, -0.379057914018631, 0.5512635707855225, 0.3605673313140869, -0.04579322412610054, 0.7864142656326294, -0.3230815529823303, 0.83531254529953, 1.0212827920913696, 0.40914443135261536, 2.3275794982910156, -0.047487203031778336, 0.6228352189064026, 0.37318602204322815, 0.7548627853393555, 0.8689396381378174, 1.1875700950622559, 0.3912874162197113, 0.41568347811698914, 0.4245399534702301, -0.033966004848480225, -0.295747309923172, -0.8021383881568909, -0.42701947689056396, 0.210132896900177, -0.1738511174917221, -1.0629408359527588, 0.8949649333953857, -1.2098337411880493, -0.27213576436042786, -1.6322917938232422, -1.2716670036315918, 0.7430375814437866, -0.9538534879684448, -0.9844955205917358, -1.2480387687683105, -2.616562604904175, -1.681438684463501, -0.028332499787211418, -1.358909249305725, -1.3771744966506958, -0.20580331981182098, 0.17994195222854614, 1.193532109260559, 0.8078858852386475, -0.7141728401184082, -1.5540931224822998, 0.36890485882759094, 0.16327665746212006, 1.3992881774902344, -1.4570444822311401, -1.5400689840316772, -0.7120983004570007, -0.45626047253608704, 0.13963067531585693, -0.8389446139335632, -1.2481286525726318, 0.23784689605236053, 0.2671297788619995, 0.9946768283843994, -1.863208532333374, 2.2089991569519043, -0.008378379046916962, -0.7001362442970276, 0.25543636083602905, -0.8587783575057983, 1.6385748386383057, -0.15909415483474731, -0.02862037532031536, 0.6179154515266418, -0.14398734271526337, -0.1858648806810379, -0.7545278668403625, -0.2298075407743454, 0.04668644443154335, -0.4367406368255615, -0.12126800417900085, 1.1629621982574463, -0.889452338218689, 0.5484063029289246, 0.8376567363739014, 0.642341136932373, -0.799897313117981, 1.0617574453353882, -0.027947667986154556, -0.7610159516334534, 0.2649350166320801, 0.9073662757873535, -1.295651912689209, 0.6449421048164368, -0.03791290894150734, 1.1513231992721558, 0.7334784865379333, -0.4003812372684479, 0.7592065930366516, 0.2916790544986725, 0.9555603265762329, 0.5128862261772156, -0.6946528553962708, -0.08648417145013809, -1.2619941234588623, -1.1954271793365479, 0.5819422602653503, 1.2950412034988403, 1.5511653423309326, -1.5030306577682495, -1.6385459899902344, -1.2755513191223145, -0.6050988435745239, -0.5762532353401184, -1.0134505033493042, 0.22106286883354187, 0.37303996086120605, 0.09642722457647324, 1.236965298652649, 0.7071118354797363, -0.34436526894569397, 0.6593518853187561, -0.6826441287994385, 0.7907326817512512, -0.42128559947013855, 0.8452625870704651, -2.5112428665161133, 0.2231922149658203, -0.24165473878383636, 1.359998345375061, -1.8810443878173828, 0.6210938692092896, -1.5687576532363892, -0.38679397106170654, 0.0761483758687973, 1.0467904806137085, -0.5546861886978149, 2.173297643661499, 1.0698868036270142, -1.553985834121704, 0.4698963463306427, -0.09808029979467392, -0.2989963889122009, 0.018124597147107124, -0.17033228278160095, 0.6203957796096802, -1.1836886405944824, 1.1117392778396606, -0.11464785784482956, 0.8617870211601257, 0.8130624294281006, -1.4465125799179077, -0.2404296100139618, 1.0904239416122437, -0.05464261770248413, -0.35010606050491333, -0.6041091680526733, 0.49087631702423096, -0.3348093330860138, 1.5293809175491333, -0.11988811939954758, -1.016972541809082, -1.339469075202942, 0.01795841194689274, -0.3151320219039917, -0.0019257526146247983, -1.228602409362793, -0.5169420838356018, -0.18341957032680511, 2.7134790420532227, -0.42382878065109253, -0.17847619950771332, 0.7452274560928345, -1.192987084388733, 0.19979815185070038, -1.1514099836349487, -0.22954495251178741, 0.5847936272621155, 0.04485771432518959, 1.0629184246063232, 0.5683735013008118, 0.9310590028762817, 1.0107783079147339, -1.0584794282913208, 0.10968927294015884, 0.6389360427856445, -0.1886848360300064, 0.7247204780578613, 0.6149165630340576, -0.6286052465438843, 0.08672334998846054, 0.35872623324394226, -0.06504286080598831, -1.8155791759490967, -0.07499909400939941, -0.48880642652511597, -0.744949221611023, 0.533686101436615, 0.2022937536239624, -0.675595760345459, 0.5748748183250427, 1.3742380142211914, 2.8134751319885254, -1.3664321899414062, -0.38335415720939636, 1.8625471591949463, -0.0908200815320015, -0.5090672969818115, -0.3109760582447052, -0.9877999424934387, -0.06595880538225174, -0.9141236543655396, -0.48651444911956787, -1.2798022031784058, -1.9533213376998901, -0.7103608250617981, -0.6921827793121338, 0.9694247245788574, -0.48821526765823364, -1.7250382900238037, -0.34911319613456726, 2.3603408336639404, -1.3159892559051514, -0.32806289196014404, 1.0604097843170166, -0.15324202179908752, -0.0671919584274292, 0.6204003691673279, 1.4044522047042847, 0.22966518998146057, -0.19995099306106567, -0.3393830358982086, 0.4823882281780243, 1.3803695440292358, -1.0858818292617798, -0.6121743321418762, -1.4627811908721924, -1.5570694208145142, -1.780203938484192, -0.35806581377983093, -0.32747164368629456, -0.8170402646064758, 0.9599445462226868, -1.1545389890670776, 0.32565397024154663, 0.05288508161902428, 1.02946937084198, -1.638195514678955, 2.373903274536133, -0.8942908644676208, -0.6543784737586975, 0.2788761258125305, -1.4863897562026978, -1.8779650926589966, 0.7065390348434448, -0.2268889993429184, -1.4802937507629395, 0.2767046391963959, -0.6373761296272278, -0.1773620843887329, 0.06392372399568558, 0.28134408593177795, 1.3333444595336914, 0.30952486395835876, -0.6680153012275696, -0.438258558511734, -0.6963364481925964, -1.2352122068405151, -0.15390872955322266, -0.4873707890510559, 0.5721396803855896, -0.13453496992588043, 0.34358811378479004, -0.06944721192121506, -0.7841594815254211, 1.2074283361434937, 0.7633761763572693, 0.08075142651796341, 0.5356301665306091, 0.9676823019981384, 1.0548982620239258, -0.5084587931632996, -0.8824874758720398, 1.4249337911605835, 0.16509069502353668, -1.3583744764328003, 1.4742774963378906, -0.8356069922447205, -0.7137624025344849, -2.693939685821533, -0.3731021583080292, 0.018079647794365883, 1.2049131393432617, -0.9494905471801758, 1.220090627670288, 0.1488293707370758, -0.1623048484325409, -1.7923812866210938, 1.7132415771484375, -0.14243851602077484, -1.5968824625015259, -1.9164361953735352, 0.8476517200469971, -0.7170501947402954, 0.07676972448825836, 0.46336492896080017, 0.1959104686975479, -0.2695120871067047, -0.6285055875778198, 0.592466413974762, -0.478620707988739, -0.10421216487884521, -0.8164476752281189, 0.48650556802749634, 0.5461683869361877, 0.3940945565700531, 2.027437925338745, -1.781252145767212, 0.8541333675384521, 1.171244502067566, 0.4962822496891022, -1.7565377950668335, -0.0756511241197586, -0.6033298969268799, -0.425028920173645, -2.5934665203094482, 0.9844785928726196, -0.47457754611968994, 0.6338001489639282, 1.155678391456604, 0.9951296448707581, 1.4917136430740356, 0.0691644549369812, 0.22325772047042847, -0.26647675037384033, -0.3667800724506378, -0.4604889750480652, 1.13200843334198, -0.1888808012008667, 0.18068033456802368, -1.0690778493881226, -0.5458657741546631, 0.5546016097068787, 0.3900952637195587, 0.10697490721940994, -0.8132901191711426, -0.5187330842018127, 1.2545015811920166, 0.389077752828598, -0.5597866773605347, -1.4065029621124268, -0.08964349329471588, -0.7778041958808899, -0.3123989701271057, -1.174927830696106, 0.21638140082359314, 0.3268990218639374, -0.1371685117483139, 0.8599926233291626, 0.18583858013153076, -0.5756564140319824, -2.218867063522339, 1.4233707189559937, -0.443239688873291, -0.9723198413848877, -0.05346839874982834, -0.01035089697688818, 0.5242742896080017, -0.213079571723938, -0.19943562150001526, -0.8486926555633545, 0.47767797112464905, -0.9048831462860107, 1.1910367012023926, 0.6852657198905945, -0.9041643738746643, 1.4478915929794312, 0.6815788149833679, 1.7796121835708618, -0.2074926495552063, 0.7969545722007751, -0.5153307318687439, -0.1981441080570221, 1.6300441026687622, 0.6528186202049255, 1.6498686075210571, -2.7774500846862793, 0.5200836658477783, -0.23950505256652832, -1.2596553564071655, 1.1025958061218262, 1.6112985610961914, 1.0243808031082153, 1.4403048753738403, -0.47308143973350525, -2.5442166328430176, -0.0633741021156311, -1.161375880241394, 2.0875043869018555, -0.4368046224117279, 0.1633685827255249, 0.04959508404135704, -0.517916202545166, 0.32969993352890015, -1.3146682977676392, 0.4693933129310608, 0.3417360782623291, -1.7197481393814087, 0.534290075302124, -2.2437052726745605, -0.26050612330436707, -0.328297883272171, 0.23051388561725616, -0.21218976378440857, 0.05748111754655838, 0.4916835129261017, 0.10401757806539536, 1.0090612173080444, -1.1180610656738281, -0.04625799134373665, -0.6058371663093567, 0.168552964925766, 1.7947956323623657, 1.106699824333191, -0.9215479493141174, 0.9315454959869385, -0.3197440505027771, -1.302081823348999, 0.7733983993530273, 2.5947906970977783, -1.5043416023254395, 1.6652135848999023, -0.9890479445457458, -0.7383314371109009, -0.5509803891181946, -0.001048246631398797, -0.34572935104370117, 0.6900564432144165, 1.3278571367263794, -0.6673586964607239, -1.042373538017273, -0.6064855456352234, 0.17446625232696533, 0.27511781454086304, -1.2500460147857666, -0.0050910660065710545, 0.3672778606414795, -1.9106866121292114, -1.4954525232315063, 0.06593883037567139, 2.379452705383301, 1.1123751401901245, -0.3531401753425598, -0.47942817211151123, 0.2724882662296295, 1.6418498754501343, 2.2153961658477783, -2.49812650680542, -0.6251581907272339, 0.003415327286347747, -0.06296831369400024, -1.0825786590576172, -0.7691473960876465, 0.3574278950691223, -0.7685134410858154, 0.6520370244979858, -0.378438264131546, -1.3742069005966187, 0.9818444848060608, 0.8154810667037964, 0.0003695568593684584, 1.010206699371338], [0.9696879982948303, -0.5948877334594727, 1.5430207252502441, -1.9932210445404053, 0.8289650678634644, -0.11233287304639816, -0.34272339940071106, -1.172389030456543, 0.3142216205596924, -0.6144468784332275, -0.08234991133213043, -1.3244026899337769, 0.9122462868690491, 0.4099728763103485, 0.09864210337400436, -1.5346375703811646, 1.2337651252746582, -0.3101809024810791, -1.363532543182373, 1.0433508157730103, -0.6358967423439026, -0.26452258229255676, 1.0221439599990845, 0.8566275238990784, -0.5479128956794739, -0.41539573669433594, -0.79615318775177, 0.9456530809402466, 0.5251405239105225, -0.6924599409103394, 1.9619089365005493, -0.495528906583786, -0.36703774333000183, -0.4528505504131317, 0.8537227511405945, -1.3861886262893677, 1.0406908988952637, 0.8512740135192871, 0.31263965368270874, -0.14624032378196716, -0.14347094297409058, 0.5768842101097107, -3.1224687099456787, -0.44060584902763367, 0.8232318162918091, 0.058089688420295715, -0.14573106169700623, 0.32549551129341125, -0.25081557035446167, -0.23824258148670197, 0.6025396585464478, 1.7434380054473877, -1.431766390800476, 0.9812753200531006, -0.9612783193588257, 0.2922498285770416, 0.06399930268526077, 0.7821898460388184, 0.5621603727340698, -0.282591313123703, 2.3897573947906494, 0.406474769115448, 0.6855259537696838, -1.5682767629623413, -0.49217548966407776, 2.388477087020874, -0.44300827383995056, 0.8826018571853638, 0.24726304411888123, 1.2015308141708374, -1.4147592782974243, -1.7386263608932495, -0.028935037553310394, -0.7692374587059021, -0.4719488024711609, -0.19577765464782715, -0.2598515450954437, 0.36993348598480225, 0.019015055149793625, -0.18742789328098297, 0.2607751190662384, 1.8314063549041748, 0.08033192902803421, 0.21270857751369476, -0.725945234298706, 0.9592713713645935, -0.24443888664245605, -0.2187582403421402, -0.34694814682006836, 1.3318947553634644, 2.07737135887146, 2.0077905654907227, -0.0991801992058754, 1.3957008123397827, 0.13663911819458008, 0.3694712221622467, -0.18728791177272797, -0.6065331697463989, 0.578121542930603, -0.9060246348381042, -1.636027455329895, 0.7563326358795166, 0.6619620323181152, 1.18271005153656, 0.554259181022644, 1.5677897930145264, 0.18978706002235413, 0.47583892941474915, -0.08480995893478394, -0.5119505524635315, -0.16667887568473816, 0.8525910377502441, -0.4143937826156616, 0.48790696263313293, 0.9659739136695862, -0.03846121206879616, 0.7181660532951355, 1.3211166858673096, 2.524704933166504, -0.05393052473664284, -0.8281831741333008, -1.564539909362793, 0.7317196130752563, 0.19693320989608765, 0.5649603605270386, 0.9679840207099915, 0.7055966258049011, 1.8004192113876343, -0.43118563294410706, 0.9516292810440063, 1.3095011711120605, 1.3488529920578003, -0.31075188517570496, 0.032673973590135574, 0.016618400812149048, -2.2131080627441406, -1.0809924602508545, 1.0589282512664795, -0.6033554673194885, 0.6341848373413086, -1.0403106212615967, 0.40447428822517395, -0.3052501976490021, -0.07354938983917236, 0.7413758635520935, -0.6614853739738464, -1.7176822423934937, 1.7083078622817993, -0.4237706661224365, 0.6369358897209167, -0.3209122121334076, -2.1983604431152344, -0.27199065685272217, 1.907964825630188, -0.8473294377326965, 0.39414066076278687, 0.8191450834274292, 0.34176772832870483, -1.1885405778884888, 1.338153600692749, 0.2940283715724945, -0.508440375328064, 0.3954222798347473, -1.086134910583496, 1.5561379194259644, 0.709577202796936, -1.7786719799041748, -0.5240187048912048, 0.8310648798942566, -0.3646482229232788, 1.1324931383132935, 0.5672031044960022, 0.17199836671352386, -1.6563472747802734, 0.4361325800418854, 0.28899016976356506, 0.8043091893196106, 0.006138606928288937, -0.4517522156238556, -1.479624629020691, -0.524643063545227, 2.055896759033203, -1.6739308834075928, 1.4092228412628174, 1.6269121170043945, 0.7864728569984436, -1.6471530199050903, 0.303264856338501, -1.9158488512039185, -1.5177942514419556, 0.4932246804237366, -1.1991095542907715, 0.6252381205558777, 0.9043100476264954, -1.1461389064788818, 0.6235209703445435, 2.159588575363159, -0.12551085650920868, -0.49412789940834045, 0.19985784590244293, -0.7146630883216858, -0.522244930267334, 1.3831475973129272, -0.09044443070888519, 0.049347471445798874, 0.41591471433639526, -0.007576784584671259, -1.5150760412216187, -1.1821062564849854, -1.788718581199646, 0.9679597020149231, 0.4737842381000519, -1.57156240940094, 1.0340633392333984, 0.18590088188648224, 0.8597253561019897, 0.2753320038318634, -0.196528822183609, -0.18296858668327332, -0.9545050859451294, -0.6010130643844604, 2.620831251144409, 0.1702696830034256, 1.5836718082427979, 0.8640979528427124, 1.2996182441711426, -1.5429354906082153, -0.24532845616340637, 0.3640824258327484, -0.8640836477279663, 1.4105464220046997, 0.09218741208314896, -0.29444169998168945, 0.22756874561309814, 0.39069080352783203, 0.21660780906677246, -1.8861936330795288, 0.08517859131097794, 0.9841690063476562, 0.7925792336463928, -0.46744441986083984, 0.7698493599891663, 0.25767216086387634, 0.5456740260124207, 1.0473878383636475, -0.4706416428089142, -0.39062321186065674, -0.7826898097991943, -0.24610339105129242, 0.06563431769609451, -0.03936086967587471, -0.9533559083938599, 0.22806517779827118, 0.20417776703834534, 0.12964734435081482, -0.42015594244003296, 0.11445777863264084, 0.8126932382583618, -1.567688226699829, -0.33888980746269226, 0.36993154883384705, 0.4708409309387207, 1.4327342510223389, -0.14238502085208893, -0.520655632019043, -1.2599520683288574, 0.7130008339881897, -0.41729044914245605, 1.2229900360107422, 1.212717056274414, -0.19034799933433533, 0.4951568841934204, 0.8259795904159546, -1.1470775604248047, 0.5225159525871277, 1.1872596740722656, -0.3881753087043762, -1.7147270441055298, -0.7578975558280945, -0.5287710428237915, -0.4245662987232208, -0.2777819037437439, -1.1821109056472778, -0.11085081100463867, 1.3282090425491333, 0.8947414755821228, -1.3934576511383057, 0.38240382075309753, -0.23292359709739685, -0.8934667706489563, -0.6930166482925415, -0.8203148245811462, 1.5352630615234375, 1.0365082025527954, -1.1559278964996338, -1.6189488172531128, -1.6367164850234985, 1.7359713315963745, 1.9776020050048828, -0.6299989223480225, -1.4391727447509766, -0.4408565163612366, -0.18088500201702118, -0.7980747222900391, 0.5164413452148438, 0.3564543128013611, 0.25554341077804565, -1.6304658651351929, 2.4694302082061768, -0.9652957916259766, 0.029760215431451797, 1.6009365320205688, -1.8757503032684326, -0.4950712323188782, 1.6737431287765503, -0.6757602691650391, 1.3706004619598389, 0.7742315530776978, -0.5552089214324951, -0.8151543140411377, -0.18568375706672668, 0.23156170547008514, -0.487112432718277, -2.11370587348938, -0.17937883734703064, -2.4634265899658203, -0.3888433873653412, -0.8360457420349121, 0.09215601533651352, -0.6015892028808594, 0.7847490310668945, -1.5435724258422852, -0.0933111384510994, -0.5057604908943176, -0.6301407217979431, -1.288061261177063, 1.4854671955108643, -1.6538212299346924, 0.2626993954181671, 1.2089848518371582, 0.5903070569038391, -1.6126863956451416, -1.2244869470596313, -0.4149792492389679, 1.351866364479065, 0.39101916551589966, 0.8232529759407043, -0.9032483100891113, -0.8327010869979858, -2.0757153034210205, 1.511106252670288, -0.21571236848831177, 0.5874793529510498, 0.42055588960647583, 0.2807084918022156, 0.3276835083961487, 0.7782264351844788, -0.9116438031196594, 0.6063229441642761, -0.16220521926879883, -0.936037003993988, -0.307976096868515, 0.10682521015405655, 0.5374422669410706, 1.2321280241012573, 0.07209748029708862, 0.3288699984550476, -0.48876136541366577, 1.370722770690918, 0.4980076551437378, 0.6536166667938232, -0.9435197710990906, 0.3992295563220978, -2.0020229816436768, 0.4013844430446625, 0.9333937764167786, 0.7385216951370239, 0.09606102854013443, -1.1652246713638306, -1.2510709762573242, -1.0599018335342407, -0.3336523473262787, -0.8338296413421631, -0.17891934514045715, -0.3833337724208832, -0.41240057349205017, 1.0281813144683838, -0.23442435264587402, 0.987955629825592, 2.1043541431427, 0.0013235623482614756, -0.2385065257549286, 0.7423384785652161, 0.042751044034957886, -0.3897445499897003, -1.1061439514160156, 0.6452707648277283, -0.6766804456710815, 0.1770104467868805, -0.18131619691848755, 0.4566626250743866, 0.39388346672058105, 1.5229130983352661, 1.2010211944580078, 0.18645954132080078, 0.497942179441452, -1.8329628705978394, 0.5015623569488525, -0.9155809283256531, 0.1686611771583557, 0.17852628231048584, -0.15719600021839142, 0.9718838930130005, 1.8899884223937988, 0.23616991937160492, -0.40657222270965576, 0.8277578353881836, -2.3875529766082764, -0.6886829733848572, 0.3052385449409485, 1.2654982805252075, -1.105748176574707, 0.3112216293811798, -0.4259926974773407, 0.24382410943508148, -0.3763616979122162, 0.4966067671775818, -0.12111825495958328, -2.3594772815704346, -0.16628167033195496, 0.3267776072025299, 0.35145536065101624, -0.19663965702056885, 1.6284894943237305, -1.362259030342102, 1.759156584739685, 0.5600895881652832, -0.9504475593566895, -0.06383451074361801, 0.1604573130607605, -2.226848840713501, -2.5148916244506836, 0.08439807593822479, 1.7902512550354004, -2.555436611175537, -0.7787429094314575, -1.8979144096374512, 0.2738983929157257, -0.744575023651123, 1.1536548137664795, -0.9477301239967346, 1.1342291831970215, -1.026449203491211, -1.7206804752349854, 1.231143832206726, -1.4543544054031372, 0.7754346132278442, -1.2715586423873901, 1.2194461822509766, 0.7765166163444519, 0.5641042590141296, 0.19030070304870605, -0.6464598178863525, 1.0518250465393066, 0.001653803512454033, 1.197226881980896, 0.3189104199409485, -0.18985751271247864, -0.8098820447921753, 0.27648788690567017, -0.4634716212749481, 0.9169362187385559, 0.019638068974018097, -1.268768310546875, -0.7647994756698608, 1.3440980911254883, -0.2842363715171814, -0.1194445863366127, 1.239658236503601, -0.17299756407737732, -1.1705241203308105, -0.7469181418418884, 0.5789840817451477, 0.625228762626648, -2.590557336807251, 0.343746542930603, -1.4860560894012451, 0.11318347603082657, 0.40440335869789124, -0.9372751712799072, 0.21494123339653015, -0.3473680913448334, 1.5771111249923706, -0.5547624230384827, -0.08479438722133636, 1.6983938217163086, -0.9539600610733032, -0.8122336864471436, -1.1556676626205444, 1.427957534790039, -0.5786747932434082, 1.2613041400909424, -1.2250272035598755, 0.8355156183242798, -2.4429144859313965, 0.08642707020044327, -0.8537917137145996, -2.2272863388061523, 0.18970684707164764, 0.009966288693249226, -0.27372264862060547, 0.366364449262619, 2.682584285736084, 1.2111706733703613, 0.5569418668746948, 0.014759618788957596, 0.05453743413090706, 1.2036820650100708, 1.607987880706787, -1.4338223934173584, -1.139721393585205, 0.6179025769233704, -1.3109029531478882, -0.08867378532886505, -0.26039692759513855, 0.24811385571956635, -1.5116186141967773, -0.6053169965744019, 1.4495787620544434, 0.5423167943954468, -0.21663042902946472, 1.1455535888671875, -1.6589645147323608, 0.1612941473722458, 0.934993326663971, 0.38926735520362854, -1.0133033990859985, 0.9723055958747864, -1.6421656608581543, -1.8912843465805054, -0.35753822326660156, -0.9856659770011902, -0.16664990782737732, -1.4519600868225098, -0.40405526757240295, 1.4667726755142212, -0.18453317880630493, -0.9156052470207214, -0.19208616018295288, 0.43572354316711426, 1.3190248012542725, 0.09036710113286972, 0.9807772040367126, 0.9067057371139526, -0.8883988261222839, -0.40180325508117676, -1.1683748960494995, 0.1535642147064209, -0.09177196770906448, -0.6639836430549622, -1.612586498260498, -0.12061956524848938, 1.1332916021347046, -0.01727345958352089, -0.24118104577064514, 0.29282596707344055, -1.6059879064559937, -1.1990505456924438, 0.11341080814599991, -0.44593578577041626, 0.8505385518074036, 0.9282820224761963, -0.5613819360733032, -0.07115485519170761, -0.44946590065956116, 0.308174729347229, 0.7667542099952698, 0.3405986726284027, -0.8537125587463379, -0.40056630969047546, -1.2470961809158325, 1.1253267526626587, -0.501767098903656, -0.23077847063541412, -0.1052340641617775, -0.46953150629997253, 0.9922837615013123, -1.5655078887939453, -1.1321579217910767, -0.5088231563568115, 0.25508779287338257, 0.12977291643619537, -0.20986950397491455, -0.060546986758708954, -0.09670433402061462, -1.0316681861877441, 2.147387742996216, -1.433976411819458, -1.6958279609680176, 0.35588544607162476, 0.2718682289123535, -1.0363972187042236, -0.4713763892650604, 1.765019178390503, -0.6113564372062683, -1.1601344347000122, -0.7688955068588257, -0.15232990682125092, 0.22164210677146912, 0.3888825476169586, 0.7266613245010376, 0.888988733291626, -1.1768642663955688, -0.5050640106201172, 0.21100939810276031, -1.7863692045211792, 1.9725099802017212, 1.566840410232544, 0.6521214246749878, 1.0498132705688477, 1.948317050933838, 0.10600874572992325, -0.39302197098731995, -1.5214424133300781, 0.9295640587806702, 1.6328656673431396, -0.43002578616142273, -1.4140493869781494, -0.9163156747817993, -0.28126582503318787, -2.1483328342437744, -0.8105567693710327, -0.0981508269906044, 0.15000198781490326, -0.603175699710846, -0.269187331199646, -1.3944575786590576, 0.2616419494152069, -0.005186745431274176, -0.4402238726615906, -1.731313943862915, 2.2078752517700195, -0.8223685026168823, 1.6521414518356323, -0.06394171714782715, -0.8138890266418457, -1.5349688529968262, 0.8142386078834534, -1.4840073585510254, 1.6356719732284546, -0.33627021312713623, 1.6300225257873535, 1.8047932386398315, 1.0656362771987915, -0.22050733864307404, 0.27542635798454285, -0.8848585486412048, -0.8654125928878784, -1.4274927377700806, 0.087853342294693, 0.6366265416145325, -0.5984776020050049, -0.30581068992614746, 1.1164063215255737, -1.0576658248901367, -2.195847511291504, -1.2644248008728027, -1.483986258506775, 0.8064257502555847, 0.43794727325439453, -1.4904688596725464, 0.8455149531364441, 0.8141880631446838, -1.1917637586593628, 1.7209980487823486, -3.01293683052063, -0.08033007383346558, -0.1474263221025467, -0.18566805124282837, -0.5760247111320496, 0.3900899291038513, 0.10548052191734314, 0.551301121711731, -0.12943868339061737, -0.6424511075019836, 0.7694863080978394, 0.8274678587913513, 0.4963567554950714, 0.14337922632694244, -0.48328763246536255, 1.2228087186813354, -0.11862718313932419, -1.0213035345077515, 0.08629342168569565, -0.5576165914535522, 0.1378907710313797, 1.324381709098816, 0.28712761402130127, -0.8271762728691101, -0.04664268717169762, 1.4815375804901123, -0.08341644704341888, -0.9954192042350769, 0.014052479527890682, 0.6530270576477051, -1.032019019126892, -1.013582468032837, -1.3964227437973022, 0.9729493260383606, 1.08818781375885, -1.2577427625656128, 1.7742464542388916, -0.7454267144203186, 1.2855018377304077, 1.2524585723876953, 0.7040137052536011, -1.0717976093292236, 0.0009499720181338489, 0.5375391840934753, -0.6218576431274414, -0.007260793820023537, -0.5751233100891113, -1.0841752290725708, 1.69052255153656, 1.4078710079193115, 1.905003547668457, 0.8413408398628235, -2.7376937866210938, -1.1313755512237549, 0.5175231099128723, -0.01392925065010786, -1.1310240030288696, -1.7076600790023804, 1.904354453086853, 0.3961026668548584, -0.5201712250709534, 1.7790865898132324, -0.3356243968009949, -0.2032841145992279, 1.124766230583191, -1.4160412549972534, -0.9645164012908936, -1.5946056842803955, -0.06277308613061905, -0.6018970608711243, -0.6849216222763062, -0.17741349339485168, -2.2025818824768066, 0.26597028970718384, -0.01924501545727253, -1.4942139387130737, 0.9941176772117615, 0.34508687257766724, -1.135515570640564, 0.1788214147090912, -0.655802845954895, 0.7472876310348511, 0.469833642244339, -0.2878187596797943, 0.4731006324291229, -2.012805223464966, -0.7764520049095154, 1.2285948991775513, -2.0503790378570557, 0.4315643012523651, 0.754831075668335, 0.5533469319343567, -1.020632028579712, -1.0613594055175781, -0.6168928146362305, 0.9952303767204285, -0.820219874382019, -0.20737485587596893, -1.1010340452194214, -0.3943099081516266, -0.0769246444106102, -0.4231026768684387, -0.31667855381965637, 1.7035276889801025, 0.35238417983055115, -1.828608751296997, 0.17581771314144135, -0.36631229519844055, -1.3293476104736328, 0.7309755086898804, 1.0862171649932861, -0.31955623626708984, -1.0712010860443115, -1.1615777015686035, -0.9927023649215698, -0.3688812851905823, -1.0600898265838623, -1.3370747566223145, -0.6725300550460815, -0.09932851046323776, 0.8208441734313965, 1.0279016494750977, 0.45403778553009033, -0.25045040249824524, 0.3528408706188202, 2.6797027587890625, -0.5954958200454712, 1.844114899635315, 0.23634959757328033, 0.018222467973828316, 0.1441381573677063, 0.6417471766471863, 0.45456185936927795, 0.128901407122612, 1.4073234796524048, 0.3870912790298462, 1.1685518026351929, -0.04789241775870323, -0.2539207637310028, 0.0541057214140892, 1.449781060218811, -0.7618741989135742, -0.6250048875808716, -0.3060745298862457, 1.1424839496612549, -0.1922876089811325, -0.1465177834033966, 0.7697203755378723, -0.8279945254325867, -0.8079437613487244, 0.5586026906967163, 0.20610487461090088, -0.5181407928466797, -0.7247483134269714, 1.561605453491211, 0.6306738257408142, -1.3168716430664062, 0.7403689622879028, -0.03495566174387932, -1.8145736455917358, 0.25571271777153015, 1.3711185455322266, -0.5727388262748718, 0.5314807295799255, 1.484947681427002, 0.36731454730033875, 0.20901328325271606, 0.09907592833042145, 0.7110852599143982, -0.6712210774421692, 1.0854274034500122, 0.00523233599960804, 0.43717125058174133, 1.2111066579818726, 0.12642614543437958, 1.9801876544952393, 0.38125765323638916, 0.4504868984222412, -1.900686502456665, 0.6639097332954407, 0.4596642553806305, -0.8575565218925476, 0.14595927298069, 0.4631918966770172, -0.8884715437889099, 0.7609630823135376, -0.8380205631256104, -0.3463420271873474, -1.0523687601089478, 0.44889315962791443, -0.525715708732605, -2.31687068939209, -1.2154555320739746, 0.940247118473053, -1.437432050704956, 0.8236269354820251, 0.1847090721130371, 0.0017935517244040966, 1.160438060760498, 0.2227081060409546, 0.3912242352962494, 0.8781638741493225, 0.25984999537467957, 0.10507063567638397, 1.7392761707305908, 0.9504179358482361, 0.1612318903207779, 0.1744455099105835, 1.7396494150161743, 0.9770851731300354, -1.5008516311645508, 1.1424082517623901, 0.19233465194702148, -0.9184815287590027, 0.19784556329250336, 0.18076777458190918, 0.19800476729869843, -1.7343555688858032, -0.02760181576013565, 0.29817813634872437, 0.6714404225349426, -0.7068020105361938, 0.5118866562843323, -1.0530222654342651, 0.08572766929864883, 0.17326921224594116, 1.1590611934661865, 0.3808855712413788, 0.7575575709342957, -0.7579262852668762, 2.298412799835205, -0.8579951524734497, 0.004717773757874966, -0.5707851052284241, -0.047432251274585724, 1.2860935926437378, -1.979182243347168, -0.25659677386283875, -1.066030502319336, 0.057372137904167175, 0.3149799704551697, -0.27848416566848755, -0.801581084728241, 0.5808208584785461, -1.6304194927215576, -0.23694656789302826, -0.2297968715429306, 1.3117002248764038, -0.7221397757530212, 1.4032189846038818, 0.7617804408073425, 0.5979216694831848, -0.055891428142786026, -0.49942946434020996, -1.0437333583831787, -1.149691104888916, -1.338547706604004, 0.7019461393356323, -0.891723096370697, 0.22663246095180511, -0.07430609315633774, -9.10101953195408e-05, 0.31142961978912354, 0.5286040902137756, 0.01818494312465191, 0.38240355253219604, 1.0907825231552124, 2.7341184616088867, 0.7167456150054932, 1.1222209930419922, 0.38560178875923157, 0.2948475182056427, -1.0181983709335327, 1.5674118995666504, -0.8815379738807678, -0.7833451628684998, 0.3611541986465454, 1.2958030700683594, -0.08426333963871002, 0.3273809850215912, -0.09990023821592331, 0.9633246660232544, 0.40865135192871094, 0.631662905216217, -0.742131233215332, -1.2246358394622803, 0.2259906381368637, -0.45772960782051086, -0.8351762890815735, 0.24940960109233856, -0.8391045331954956, 1.4828397035598755, -0.05587853863835335, 0.7419386506080627, 1.5326731204986572, 0.7374204397201538, -0.22291669249534607, 0.7942863702774048, 0.36340832710266113, 0.27393797039985657, -0.345041424036026, 0.7544010877609253, 0.35441628098487854, -0.6903352737426758, -0.12530416250228882, -1.3416041135787964, 0.7839745879173279, 2.3820347785949707, -1.866422176361084, -0.9960317015647888, 0.8950045704841614, -0.041090503334999084, -2.9847166538238525, -0.29798224568367004, -0.9253097176551819, 1.4600026607513428, 0.37404775619506836, 0.3059435188770294, -0.6061809062957764, -1.4303678274154663, -1.1068207025527954, 1.1415706872940063, -0.359836608171463, 0.9033340215682983, 0.28540369868278503, -0.4060782790184021, 0.224679097533226, -1.0112786293029785, -0.07547003030776978, -0.32742881774902344, -1.2387043237686157, -0.9778695106506348, 0.5851593613624573, 0.5315726399421692, -0.0557631179690361, 0.10992532968521118, -0.7857304811477661, -0.0216514952480793, 0.06302765756845474, -0.40965062379837036, 0.7313969731330872, 1.3538252115249634, -0.9270424842834473, 0.8491361141204834, 1.136859655380249, 0.3169858157634735, -1.9366047382354736, 0.20278982818126678, -0.022071918472647667, 0.11607713997364044, -0.4254074692726135, -0.22015602886676788, -0.21524719893932343, 0.34837502241134644, 0.5061982870101929, 0.23987318575382233, 0.4314761757850647, -0.8838452100753784, 1.0387669801712036, 0.7842273116111755, -0.2449944019317627, -0.4546375870704651, -0.6653532385826111, -0.09562098234891891, -0.3966456651687622, 1.3120378255844116, 1.2846901416778564, 0.602642834186554, 0.604257345199585, -0.23912347853183746, 1.2113969326019287, -0.4712935984134674, 1.27549147605896, 1.228407621383667, 0.298016756772995, 1.2792961597442627, 1.647452712059021, -0.7260147333145142, 0.6763384938240051, 0.03767141327261925, -0.5345492362976074, 0.9167225360870361, -1.905371069908142, 0.8447445034980774, -0.7168354988098145, 1.8853933811187744, 1.0937644243240356, -0.16776026785373688, 1.0101264715194702, -1.2660019397735596, 0.095818892121315, -0.11923360079526901, 0.21069476008415222, 0.2419121265411377, 0.09508204460144043, -0.18159276247024536, 0.5525159239768982, -0.1736522614955902, -0.7320507764816284, -1.7289094924926758, 1.5136737823486328, 1.0814400911331177, -0.3335394561290741, 1.1735814809799194, 1.869676113128662, 1.4240506887435913, 0.4182687997817993, 1.0215342044830322, 1.3587850332260132, 1.5115468502044678, -1.1379923820495605, 0.45307615399360657, -1.0802265405654907, -1.053539752960205, -0.580529510974884, 0.09512608498334885, -1.5318882465362549, -1.4190163612365723, 0.07891678810119629, 1.2365875244140625, 0.18851706385612488, -0.3390284776687622, -0.8460855484008789, -0.6588734984397888, -1.1664447784423828, 2.097470760345459, 1.2148170471191406, 0.42087870836257935, 2.49027419090271, -1.3222522735595703, 1.5942347049713135, 1.5480011701583862, -0.3385096490383148, -1.024815320968628, 0.08823171257972717, -0.38035154342651367, 0.7954931259155273, 1.2481374740600586, 0.6144508719444275, 0.30599844455718994, 1.3675802946090698, 0.3917756974697113, -1.0258344411849976, 0.23922452330589294, 1.7016180753707886, 0.42307355999946594, -0.3347458243370056, 0.49087169766426086, 1.8662322759628296, 0.5482475757598877, 0.5766382813453674, 0.9847940802574158, 0.9705456495285034, 0.8728249073028564, -0.6741747260093689, -0.06522715091705322, -0.6714591383934021, -0.4581189453601837, 0.30329564213752747, 0.9630514979362488, -0.05518738180398941, -1.359635353088379, -0.09699308127164841, 1.920689582824707, 0.008044794201850891, -0.8699445128440857, 0.49954140186309814, 0.07362309098243713, -0.13391008973121643, 0.7541250586509705, -0.8685197234153748, -1.5163768529891968, 2.672745704650879, -1.7918089628219604, 0.981592059135437, 1.5200086832046509, 0.30468541383743286, 0.29963675141334534, -0.8268015384674072, 1.2151480913162231, 1.758966326713562, -0.6457761526107788, -0.513552725315094, 0.6478553414344788, 0.963546633720398, 0.00011802239896496758, 0.18677200376987457, -0.3817693591117859, 0.4950556457042694, -0.16695867478847504, -0.15335650742053986, -1.1674927473068237, -1.1005553007125854, -1.0738192796707153, 0.037689171731472015, 1.2848719358444214, -0.5061492919921875, -0.3820897936820984, -1.410136342048645, -1.3250504732131958, 0.021462738513946533, -0.30318206548690796, 0.5992878079414368, 0.36489811539649963, 2.4777657985687256, -0.19266943633556366, 0.2039971500635147, -1.1553466320037842, 0.3365814685821533, -0.18291546404361725, -0.46783778071403503, 0.8047330379486084, 0.41638803482055664, 1.2948018312454224, 1.9291144609451294, -0.9062867760658264, -1.4823535680770874, -0.3167349398136139, 0.8080533742904663, 0.2554488778114319, -0.5535569787025452, -0.7743794322013855, 0.47632306814193726, -0.7211197018623352, -0.8468011617660522, -0.9022165536880493, -0.4421893358230591, -0.2727770209312439, 0.5541673302650452, 0.2351258546113968, -0.9603722095489502, -1.152651071548462, -0.30428794026374817, -0.5864428281784058, 0.21853245794773102, 0.4746937155723572, 1.7330207824707031, 1.173553228378296, 0.30928120017051697, 0.5104513168334961, -0.050317686051130295, 0.2401878833770752, -0.6229442954063416, 0.09953979402780533, 0.45797282457351685, 0.42415857315063477, 0.07369454205036163, 0.8008941411972046, 0.8018917441368103, -0.15104882419109344, -0.5425220727920532, 0.23117129504680634, 0.6370586156845093, 1.8604168891906738, -0.6509881615638733, 1.4077664613723755, 0.591415524482727, 0.686383068561554, 0.7530868649482727, 0.2949313223361969, 0.41823697090148926, -0.6322723627090454, -1.0962457656860352, -0.20640917122364044, -1.054503083229065, -0.7084490060806274, 0.9694058299064636, -1.0273202657699585, 0.38540780544281006, 0.2637892961502075, -2.9309396743774414, -1.3391886949539185, 0.24612769484519958, -0.5611812472343445, -1.2759662866592407, -1.738183856010437, -0.19555923342704773, -0.008086764253675938, 0.9939209818840027, -0.917625367641449, 0.07354715466499329, 2.21461820602417, -0.6387324929237366, -0.38913917541503906, -1.3923019170761108, -1.659582257270813, 0.25218915939331055, 1.126057505607605, -0.05877590924501419, -0.08890100568532944, 0.5446717143058777, -0.19117292761802673, 0.1262529045343399, -1.1809769868850708, -0.19935616850852966, -1.4179545640945435, -0.41703659296035767, -0.9468274712562561, 0.17250196635723114, 0.9003942012786865, 0.4889276325702667, -0.07649869471788406, -0.10059506446123123, 1.6287873983383179, 0.12962418794631958, -0.7972092032432556, -0.8367069363594055, -0.767153263092041, 0.3504214882850647, -3.088937997817993, 0.11224659532308578, -1.5645396709442139, -0.7496448755264282, -0.09349918365478516, -0.27755147218704224, 0.00026760855689644814, 0.7186422944068909, -1.9759547710418701, 0.23413577675819397, -0.06338127702474594, -0.38439252972602844, 0.02145608328282833, 1.014839768409729, 2.1070609092712402, 0.9241929650306702, -0.21023815870285034, -0.03147092089056969, -0.04491990804672241, 2.0806772708892822, 0.8388354182243347, -0.11959230154752731, 0.9302775859832764, -0.30544590950012207, 1.8465156555175781, -0.805335283279419, 1.8822687864303589, -0.4935730993747711, 0.05246153101325035, -1.3166439533233643, -0.5479817986488342, 1.2904305458068848, -0.4898298978805542, -0.5339193940162659, -0.26172196865081787, -0.8456873297691345, -1.1956768035888672, 0.16546641290187836, -0.6276174187660217, 0.5276662111282349, -0.43474704027175903, 1.1965521574020386, 0.43426811695098877, 1.6859424114227295, 0.7514024376869202, 0.33584779500961304, -0.8475509881973267, 0.02057052031159401, 0.1569412350654602, 1.4566560983657837, 1.0923222303390503, 0.7129734754562378, 1.9924449920654297, -1.8419396877288818, 0.6114763617515564, -0.6998757123947144, 2.089704990386963, -1.2194950580596924, 0.1359119713306427, 0.44376590847969055, 1.2583345174789429, 0.043976109474897385, -1.351686954498291, -1.0163391828536987, 0.1374116837978363, 0.7041248679161072, -1.4904465675354004, 0.6364693641662598, 0.9300070405006409, -0.14326117932796478, 0.49634742736816406, 0.9013074040412903, 0.11471480876207352, 0.034721456468105316, 0.913844883441925, 0.6406576037406921, -0.4121788740158081, 0.6841332912445068, 0.7914264798164368, 0.14603526890277863, -0.9439556002616882, 0.6273266673088074, -1.033179759979248, 0.30545833706855774, 0.263578861951828, -0.19290746748447418, -0.6465483903884888, -0.4328314960002899, 1.3989319801330566, 0.6090878248214722, -0.4985743761062622, -0.7046378254890442, 0.8588829636573792, -0.3419557809829712, -1.6054593324661255, -1.4719656705856323, 0.4632362723350525, -1.4406776428222656, -0.36166220903396606, -0.7316349744796753, 0.7358916997909546, 0.91306471824646, -0.1888262927532196, -1.4504899978637695, -0.5454221963882446, 0.8899974822998047, -0.2919712960720062, 0.9863013029098511, -1.5344717502593994, 1.5105408430099487, -1.5626713037490845, 0.9652763605117798, -0.5336992144584656, -0.5548791885375977, 0.07351045310497284, 1.2010395526885986, -0.15192805230617523, -0.4832584261894226, 2.2782094478607178, 0.6634730696678162, -0.9619861245155334, -0.8564246296882629, 0.16454680263996124, 0.39724525809288025, -1.7212952375411987, -0.7598056197166443, -0.7395023107528687, 0.9422680139541626, 0.009832441806793213, 1.5342365503311157, -0.46789684891700745, -2.3765454292297363, 0.3157302141189575, -0.2127630114555359, -1.05728280544281, -0.872896134853363, 0.10360566526651382, 0.23559963703155518, -1.5060009956359863, 0.8502399921417236, -0.12406259030103683, 0.20048010349273682, -2.003743886947632, 0.5103932023048401, -0.6578142046928406, 0.6198625564575195, 1.6946321725845337, 2.2460591793060303, -0.2795705795288086, -2.3515443801879883, 1.1350218057632446, 1.7566500902175903, -0.5171242356300354, -1.4312528371810913, 0.19762517511844635, -0.11621512472629547, -0.8430367112159729, 1.5394011735916138, 0.2065090388059616, 1.078161358833313, 0.23547819256782532, -1.0277522802352905, -1.277742862701416, -0.9149284362792969, 2.212785005569458, 0.6327153444290161, -0.041987042874097824, -0.17325814068317413, 1.2612547874450684, -0.43275654315948486, 0.4271729588508606, 0.3494953513145447, -0.43542081117630005, 0.2546551525592804, 1.034006118774414, 1.2592352628707886, -0.3528693616390228, -0.8265895247459412, 0.2864569127559662, 0.20287124812602997, -0.37717923521995544, -0.14293740689754486, -0.7833130955696106, 0.13581973314285278, -0.46478044986724854, 0.695793867111206, 0.5816047191619873, -1.5175482034683228, 0.19472604990005493, -0.4099003076553345, -0.9366158246994019, 0.5239181518554688, 0.08379963040351868, 1.2572954893112183, -0.5555177927017212, 0.25339019298553467, 1.4053916931152344, 0.7730682492256165, 0.5934435129165649, 0.31486549973487854, 0.8844709992408752, -0.244394913315773, -0.7098204493522644, 0.16716982424259186, -3.2584261894226074, -0.5277463793754578, -0.5738891959190369, -1.7643901109695435, -2.2648653984069824, 0.6353516578674316, -0.4504030644893646, 0.018278028815984726, 0.2052941769361496, 0.32864081859588623, -0.4441337585449219, -0.9422603249549866, -0.7340000867843628, -1.3146021366119385, -1.6420999765396118, 0.7041512727737427, 2.043890953063965, -0.45230868458747864, 0.35919561982154846, -0.7432669401168823, -0.03540346026420593, -0.14671191573143005, -0.7348085045814514, -0.11121996492147446, 0.4923586845397949, -0.028300944715738297, 0.8518080711364746, 0.6697766184806824, -1.8495018482208252, 1.6672712564468384, 0.9444161057472229, 0.6624342799186707, 0.2063034474849701, -0.42781928181648254, 0.20858410000801086, -1.1217763423919678, 0.26924023032188416, 0.051455408334732056, -0.9011332988739014, 0.2590402662754059, 0.480436235666275, -1.016469955444336, 0.8720410466194153, -1.7888931035995483, -0.04476574435830116, -0.044387318193912506, -0.012592662125825882, 2.799229383468628, 0.8984850645065308, -0.3252837359905243, -0.09666486829519272, -0.2770061194896698, -1.3765878677368164, -1.6193487644195557, 0.8105029463768005, 1.122527003288269, -0.49207964539527893, 0.33992621302604675, -0.7241904139518738, -0.18970166146755219, -0.29519858956336975, -0.23646456003189087, -1.3367537260055542, -0.15995773673057556, -0.685924768447876, -0.31366464495658875, 0.1534912884235382, -0.21107473969459534, 2.079118490219116, -0.9777989387512207, 0.478569895029068, -0.790108323097229, 0.4643545150756836, 0.5033639073371887, -0.07250330597162247, 0.6104884743690491, -0.17196284234523773, 1.6735213994979858, -0.8169301152229309, 0.3769838213920593, -0.29606202244758606, 0.39231568574905396, 1.0009819269180298, -0.7588160037994385, 0.8857430219650269, -0.6558279991149902, -0.9725530743598938, -2.0829856395721436, -0.4083751440048218, -0.006978982128202915, -1.0866076946258545, -0.9630460143089294, -0.6751781105995178, -2.6300528049468994, 1.5245561599731445, 0.019914165139198303, 0.6292248368263245, 0.5602970719337463, -1.3043584823608398, -0.40383753180503845, -0.01979980804026127, -0.6353054642677307, 0.08972474932670593, 0.8916540145874023, 1.7713096141815186, 0.477937787771225, 0.30450132489204407, -1.3441678285598755, 0.8316695094108582, 0.6863425374031067, 0.31224384903907776, -0.06591511517763138, -0.30333346128463745, 0.8627716302871704, 0.0248735211789608, 0.4171529710292816, 1.0690431594848633, -0.2485860288143158, 0.5072341561317444, 1.363339900970459, 2.0639734268188477, 1.8811578750610352, -1.7822635173797607, -1.5431315898895264, -0.13158346712589264, 0.7855671644210815, 0.580041229724884, 1.1798263788223267, -1.033219337463379, 0.6032026410102844, 0.09698404371738434, -1.2245142459869385, -0.24270544946193695, 1.83056640625, -0.5095301866531372, 0.9821353554725647, -0.16878364980220795, -0.2331111878156662, 2.151000738143921, 1.1996879577636719, -0.941044807434082, 1.4872610569000244, -0.41383683681488037, -2.2645063400268555, 0.18725267052650452, 1.3001978397369385, 0.5881398320198059, 1.949747920036316, -0.6777576208114624, -0.1015077456831932, 0.394582599401474, 0.2891712188720703, 0.1338500678539276, -0.21137435734272003, -1.1379252672195435, 0.36560726165771484, 0.5843451619148254, -0.6951500177383423, -0.397095650434494, -1.8570771217346191, -1.9010976552963257, -1.8518346548080444, -0.8898884654045105, -0.7500206828117371, -1.8646166324615479, -0.5743189454078674, -0.6999656558036804, -1.4426920413970947, -0.9219735860824585, 0.0022654675412923098, -1.5413689613342285, -0.7983105182647705, 0.37061837315559387, 1.464141607284546, -0.01514384150505066, 1.4415346384048462, 0.0796087384223938, -2.027381420135498, -1.0148744583129883, -0.5296804904937744, -0.2725086212158203, 1.232822299003601, -0.7539992928504944, -1.277705430984497, 1.7112905979156494, -1.1405630111694336, -0.8980674147605896, -0.2664238214492798, 1.664280891418457, 0.7072256803512573, -0.4527451694011688, -0.3148724436759949, 0.6854496002197266, -1.2674931287765503, -1.0963618755340576, -0.8584434986114502, -1.0880428552627563, -1.1158738136291504, 0.678545355796814, 0.6476478576660156, -2.054962396621704, 1.101661205291748, -0.6323416233062744, -0.17097152769565582, 1.3135066032409668, -1.2565184831619263, 1.2932384014129639, 0.28170475363731384, -1.6518267393112183, -0.004501108080148697, -2.2196903228759766, 0.06671744585037231, 1.85814368724823, -0.34854111075401306, -1.3882249593734741, 0.8635414242744446, 0.4176501929759979, 1.0544618368148804, -0.13083568215370178, -2.8554487228393555, -1.9967867136001587, 0.2566010653972626, 0.4845184087753296, 1.1127334833145142, 0.4298349618911743, 0.13248798251152039, -1.3355315923690796, -0.4323112964630127, 0.8890862464904785, -1.0426732301712036, -1.3329048156738281, 1.6533973217010498, -1.7974570989608765, 1.8481254577636719, 0.010607318952679634, 0.19320133328437805, 0.5478234887123108, 0.9869384169578552, 0.6902067065238953, -0.0014418723294511437, 2.9917056560516357, -0.8979178071022034, -0.47388413548469543, -0.10583975166082382, 1.9554030895233154, -0.0756848007440567, -0.498854398727417, -2.25758957862854, -0.3994917571544647, -0.44955548644065857, 0.6470907926559448, 0.08827602863311768, 0.4251495599746704, 1.4235919713974, 0.7354801297187805, 0.6778481006622314, 0.6668644547462463, 0.16324003040790558, -1.468824028968811, 1.3434854745864868, -1.2010133266448975, -1.2481564283370972, 0.7641389966011047, 0.24249336123466492, 0.15646077692508698, -1.4123892784118652, 0.43239685893058777, 0.7920225262641907, 0.6382547616958618, -0.6279038786888123, 1.0834254026412964, -1.5568310022354126, 0.8342620134353638, -0.2642163038253784, 0.47001776099205017, -0.901355504989624, -1.1702311038970947, 0.5009081363677979, 0.33256304264068604, -2.081660270690918, 1.180738091468811, -0.4338509440422058, 2.75056529045105, -1.5949437618255615, 1.2196083068847656, 0.6550096869468689, 0.1732117384672165, 0.6011385917663574, 0.49920928478240967, -0.2686796486377716, -0.9529901742935181, -1.9077458381652832, 0.06363996118307114, 0.09208331257104874, -0.757102370262146, -0.8860362768173218, -0.1636447161436081, 1.1372102499008179, -0.045801881700754166, 1.1500356197357178, -1.7137086391448975, 1.1917039155960083, 0.5839354991912842, -1.1508948802947998, -0.9249263405799866, -0.035305798053741455, 0.9469793438911438, -0.01646721549332142, 0.8681336045265198, -0.18689052760601044, -1.712454915046692, 1.221982479095459, 0.9017845988273621, -0.44549882411956787, 0.20803824067115784, -1.1724796295166016, 1.367691993713379, 0.6488863229751587, -1.4870884418487549, 0.5881365537643433, 0.09998130053281784, 0.953865647315979, -0.35203102231025696, 0.21782691776752472, -0.006827919278293848, -0.03540503606200218, 0.4530743658542633, 0.34636586904525757, -0.1572936326265335, 0.4282582104206085, 1.1803364753723145, -1.1080650091171265, 1.2000586986541748, -0.44470056891441345, 0.40604379773139954, -0.4674798548221588, 0.5847489833831787, -0.7464117407798767, 2.1372220516204834, -0.11881285160779953, 0.3598453402519226, -1.4458935260772705, -1.5081058740615845, 0.3351280987262726, 0.5172230005264282, 1.125732660293579, 0.3855307102203369, -1.3376611471176147, -0.3725225031375885, 0.9433608651161194, -1.5986677408218384, 0.8575088977813721, 1.1802432537078857, 0.5118990540504456, -1.0337834358215332, 0.15560749173164368, 1.138393521308899, -0.5500892996788025, -0.29204487800598145, 0.8487135767936707, 0.9271007776260376, 0.6657681465148926, 0.3660956621170044, -0.6893815398216248, -0.38367006182670593, 0.3484610319137573, -1.0959302186965942, 0.38841649889945984, 0.4159221947193146, -0.5611485838890076, 0.5362919569015503, -0.7805727124214172, 0.570207417011261, 1.66506826877594, -1.8908973932266235, -0.6028265953063965, -0.7313169240951538, 0.20526212453842163, -1.5304185152053833, 0.3122512400150299, 0.15095287561416626, 0.437416672706604, -1.0975600481033325, 0.5566399097442627, -0.26235127449035645, 0.2585127055644989, -0.5492003560066223, -0.5480552911758423, 0.3518919050693512, -1.5457618236541748, -0.7483808398246765, 0.010275948792696, 0.6656513214111328, -0.8949036598205566, 0.29206937551498413, -0.6544546484947205, 0.8164030909538269, -0.2569747865200043, -1.2660353183746338, 0.1207030788064003, 0.013606484979391098, 0.8620548844337463, -0.30565398931503296, 0.28102385997772217, -0.5067719221115112, 1.3406888246536255, -1.8225044012069702, 0.6451958417892456, 0.45978546142578125, -0.995323121547699, 0.3152414560317993, 0.8816158175468445, 0.2847682237625122, -0.8193361163139343, -0.31328892707824707, -0.7404950261116028, 0.05884113535284996, -0.3235747516155243, -0.7511046528816223, -0.7979426980018616, -0.1915416568517685, -1.2191463708877563, -0.746637225151062, -0.5189638137817383, -1.1029126644134521, -0.5370946526527405, -1.178939938545227, -0.6209914088249207, -0.0716165229678154, -1.3173511028289795, -0.839043378829956, -0.44773218035697937, 0.784292995929718, 1.6648913621902466, -1.3362057209014893, 1.1549279689788818, 0.9400604963302612, -0.37829744815826416, -1.1619054079055786, -1.1065593957901, -0.016293011605739594, -0.6498944759368896, -1.2209205627441406, -1.9724030494689941, 0.323839396238327, -1.0326621532440186, 0.6327632069587708, 1.6146697998046875, -0.5153446197509766, -0.08784189820289612, -0.32850292325019836, 0.771103024482727, 0.8491606712341309, -0.0275412630289793, 0.787708044052124, 0.9953592419624329, 0.9147438406944275, 0.8477188348770142, 0.9997552633285522, 0.44065025448799133, 0.6977439522743225, 0.8965549468994141, -1.0949257612228394, -2.223670482635498, 1.1458642482757568, -2.0332164764404297, 1.2118877172470093, -1.332092523574829, 1.6707634925842285, 0.03593122959136963, -0.3673110008239746, 0.15511327981948853, -1.2498321533203125, 0.9190027713775635, 1.5109853744506836, 0.368425190448761, 1.610855221748352, 1.4206016063690186, 2.443777322769165, 1.653482437133789, -1.3885551691055298, -0.2113371342420578, -0.076836958527565, -1.0651427507400513, -0.6563759446144104, 1.0955665111541748, 0.2697369158267975, -1.0982873439788818, -0.4673323333263397, 0.4516561031341553, 1.0270025730133057, -0.5627550482749939, 0.8179206252098083, -0.8398066759109497, -1.1278728246688843, -0.5962072610855103, -0.7049278616905212, -0.4541032314300537, -0.12549638748168945, 0.2242361605167389, -1.1281589269638062, -0.24730104207992554, 0.43192312121391296, 0.07238121330738068, 0.6517958641052246, -0.4345276355743408, -0.03264299035072327, 1.013791799545288, -0.7568280100822449, -0.46484366059303284, 1.6432230472564697, -0.6183319091796875, 0.06469637900590897, -0.9696466326713562, 0.08610130846500397, 1.5233755111694336, 0.5168229341506958, -0.5630276799201965, -0.32997554540634155, 0.22822195291519165, -0.5156564116477966, 0.7464662790298462, -1.5683677196502686, -1.5478873252868652, 0.4823416769504547, 0.5056772232055664, 0.29230448603630066, 1.364579677581787, -2.4314534664154053, 0.3930527865886688, -0.004492033738642931, -0.348061203956604, 0.9106738567352295, 1.055991530418396, 0.6614327430725098, 0.932145357131958, -0.039971813559532166, -1.658424973487854, 2.113250970840454, 1.4198832511901855, 0.5830320119857788, 1.4931244850158691, -0.9193474054336548, -0.004374528303742409, -0.6007652282714844, -1.1834421157836914, -0.5897128582000732, -0.6370768547058105, -0.6828294396400452, 0.8693879842758179, -1.3646758794784546, 0.0055992985144257545, -1.8513202667236328, -1.3034340143203735, -0.42159929871559143, 0.4504390358924866, -0.48504966497421265, -1.0208977460861206, 0.8700759410858154, -0.2917236387729645, 0.13990308344364166, 0.3978074789047241, 0.1501636505126953, 0.13308365643024445, -1.114830493927002, 0.85426926612854, 3.544802665710449, 2.1723389625549316, -1.9371660947799683, 0.8594525456428528, 1.8439017534255981, 1.632893681526184, 0.23608051240444183, -1.154934287071228, -0.9023321866989136, 0.33226993680000305, -0.796623170375824, 1.5439194440841675, 1.3745672702789307, 0.3226720690727234, -0.06849893927574158, -0.30573129653930664, -0.746601402759552, 0.3910447657108307, 0.08354402333498001, -0.5562431812286377, -1.6431375741958618, 1.1051002740859985, 0.7033380270004272, 0.717616081237793, -0.6172041893005371, 0.6191536784172058, -1.72321617603302, -0.5701397657394409, 0.9463244676589966, -0.7081565260887146, -0.21126846969127655, -0.04854293912649155, -0.3085874021053314, -0.4049314260482788, 0.689306914806366, 0.08021196722984314, 1.7726995944976807, -0.7156329154968262, 0.12081358581781387, -1.8879895210266113, -0.3752502501010895, 1.6392273902893066, 0.17791837453842163, -0.4479845464229584, -0.24483171105384827, 0.5706183314323425, -2.1448514461517334, 1.516535997390747, -0.09092666208744049, -0.8105781674385071, 1.6963374614715576, -1.749741554260254, 1.7736796140670776, 1.149064064025879, -0.7942481637001038, -1.9084750413894653, -0.41913336515426636, 1.3167895078659058, -1.8982422351837158, -0.2898845672607422, 0.3983781337738037, -0.31165552139282227, -0.5521453619003296, 0.19261212646961212, -1.2959626913070679, -0.4462892413139343, 0.9141874313354492, 1.951599359512329, 0.6813660264015198, 0.9072195887565613, -1.7760136127471924, -0.31973350048065186, 0.18386177718639374, -0.98513263463974, -0.42568591237068176, 0.6738311648368835, -0.6444446444511414, 1.211530089378357, 1.4181623458862305, -1.2324498891830444, -0.5630785822868347, -1.2880299091339111, -0.3409503698348999, -0.13597339391708374, 1.3197121620178223, -0.5722538828849792, -1.3192379474639893, -0.835088312625885, 1.1618037223815918, -0.16415828466415405, 0.061142370104789734, 1.1971584558486938, -1.0706623792648315, 1.1131465435028076, 1.528045415878296, -0.24749961495399475, 0.5488210320472717, 0.5333590507507324, -0.28061601519584656, 0.7781273722648621, -0.1435827910900116, -1.7780632972717285, 0.6878049969673157, 0.4390887916088104, -0.5905547142028809, -0.1451546996831894, -0.2521609961986542, 0.9479840397834778, 0.6199293732643127, 0.5807692408561707, 0.2917172610759735, 0.43595096468925476, -1.627489686012268, -0.2967783808708191, -0.9653513431549072, 0.8793544173240662, -1.476235270500183, 2.7938740253448486, -1.3135579824447632, 0.6694203615188599, 0.3343348503112793, -0.006717171519994736, -1.4052326679229736, 2.1670196056365967, -0.0760750025510788, 0.7012491226196289, -0.9453780055046082, 0.19766592979431152, 2.8411049842834473, 0.4573350250720978, -0.982905387878418, -0.6988832950592041, 0.4431167542934418, -0.34675049781799316, -0.7310147285461426, -0.23716393113136292, 0.5817084908485413, -1.194018006324768, 0.19734857976436615, 0.878284752368927, -1.9374605417251587, 1.0841096639633179, 0.3179456293582916, -0.4559800624847412, 1.8865373134613037, 1.663336157798767, 1.1799321174621582, 1.233201503753662, 0.47410696744918823, -0.22064876556396484, -0.798986554145813, -1.6201311349868774, 0.3537547290325165, -0.9571784734725952, -1.9600785970687866, -0.06181269884109497, 0.6878042817115784, -0.19269342720508575, -0.350477010011673, -0.8606385588645935, 1.1021603345870972, -1.4652493000030518, -0.7023875117301941, 0.5730331540107727, 0.2696608006954193, -1.4723330736160278, 0.19744516909122467, 0.14143764972686768, 1.8010183572769165, -1.791796088218689, -0.8586370944976807, -1.2169102430343628, -1.5934576988220215, -0.7008548974990845, -0.547340989112854, -2.79856538772583, -0.6231555938720703, 1.5390352010726929, -0.053044047206640244, 0.21757234632968903, 1.1469134092330933, 0.011586223728954792, -1.1423333883285522, 0.5305763483047485, -2.1070945262908936, 0.13407345116138458, -0.699680507183075, -0.6194537281990051, -0.24904674291610718, -1.072004795074463, -1.1244083642959595, -1.2682496309280396, -0.3703034520149231, 2.5132863521575928, -0.6877566576004028, -0.18705952167510986, -0.2312370389699936, 0.33461350202560425, -0.2981465756893158, 0.23340094089508057, 0.8888617753982544, 0.914564847946167, 0.4787363111972809, -0.6894596815109253, -0.5191622972488403, -0.615074872970581, 0.0852278620004654, -0.6858075261116028, 1.5149823427200317, -0.22460773587226868, 0.7577795386314392, 1.955333948135376, 2.260935068130493, -0.7894863486289978, -1.0135223865509033, -1.9642866849899292, 0.2490386962890625, -0.29950132966041565, -1.470112919807434, 0.7450180053710938, -0.8271400928497314, -1.420181155204773, 0.1701509803533554, -1.7929511070251465, 0.16419284045696259, 0.30614492297172546, -0.8818769454956055, 1.5947284698486328, -0.7128564715385437, -0.622226893901825, 0.3905712366104126, -1.2059381008148193, 0.78249591588974, -0.7500389218330383, -1.0010348558425903, -1.510216236114502, -0.11220266669988632, -0.8251012563705444, -0.4518013000488281, 0.017281951382756233, 1.5457587242126465, -0.4180561304092407, 0.5355880260467529, 0.3770081400871277, -1.1452150344848633, -0.2069728672504425, 0.5851542949676514, -0.25272732973098755, 1.0775550603866577, 0.22693347930908203, 1.0660327672958374, -0.05106176435947418, 0.3654868006706238, 0.44011348485946655, -0.16497622430324554, 0.06886817514896393, 0.8387616276741028, 0.430001437664032, -1.7612905502319336, -1.0638186931610107, 1.347975254058838, -0.5620030760765076, 0.25674235820770264, 0.3242359459400177, 1.5869594812393188, 0.008677260018885136, -0.17728812992572784, -1.3673534393310547, -1.1221020221710205, 0.8597489595413208, -0.5903713703155518, 0.7704781293869019, 0.501920759677887, 1.398230791091919, -0.2792001962661743, -0.2140377312898636, -0.7439671158790588, -1.133731722831726, -0.5743463039398193, 1.7421246767044067, 1.2136142253875732, -1.9559794664382935, 0.773404061794281, 2.2128658294677734, 0.3288072347640991, -0.3858877122402191, 0.046995390206575394, 0.3578617572784424, 1.781737208366394, 0.879611611366272, -1.7900786399841309, -0.9195627570152283, 0.9220048785209656, -1.4292330741882324, 0.9055185914039612, 0.548747181892395, -0.1120673343539238, -0.033178284764289856, 0.7911694049835205, 0.3214140236377716, -0.31179314851760864, -0.024238504469394684, 1.4834319353103638, 0.1543201357126236, 2.0398006439208984, -1.6411323547363281, 1.0439273118972778, 0.14559681713581085, 0.4316193461418152, -0.5948046445846558, -1.271727442741394, -1.6566051244735718, -0.0236977506428957, 0.8041830658912659, 0.6252023577690125, 0.33414843678474426, 0.19529825448989868, 0.8004851937294006, 0.04345807433128357, 1.4509836435317993, -0.8411967754364014, 0.967863142490387, -0.46542540192604065, -0.8336730003356934, 0.11965376883745193, 0.6519154906272888, 0.20026198029518127, -1.265955924987793, 0.8884543180465698, 0.8614450693130493, -0.1830921471118927, -0.28154680132865906, 0.07774189114570618, -1.3351296186447144, -0.18943801522254944, 1.2353718280792236, 0.8850526213645935, 0.2442958503961563, -1.1147408485412598, -0.7238436937332153, -0.5447007417678833, -0.7400457859039307, -0.013178753666579723, -1.2621020078659058, -1.5817197561264038, 0.5456722974777222, 0.7473388314247131, -0.1505802869796753, 1.4925752878189087, 0.8837418556213379, -0.268492192029953, 0.07705076038837433, 0.9084384441375732, 0.21305666863918304, 0.2697577476501465, 2.2449421882629395, -0.5915483236312866, 0.02598041109740734, 1.8663190603256226, 0.22112570703029633, 0.3991040289402008, -1.0826759338378906, 1.6022844314575195, 0.31975165009498596, 1.330852746963501, 0.7726818919181824, -1.4903820753097534, 2.3876359462738037, 1.2242194414138794, 0.17373165488243103, -0.9577491879463196, -0.9486648440361023, -0.25776615738868713, -0.20094314217567444, 0.5028842091560364, 0.2030819207429886, -0.05279321223497391, 0.6204717755317688, -0.15430565178394318, 0.21685539186000824, -0.01986885629594326, -0.3358728587627411, -0.199166938662529, 0.7226229310035706, 0.5617087483406067, 0.5422030091285706, -1.3721137046813965, -2.375317096710205, 0.9002093076705933, 0.6995083689689636, -0.24659039080142975, 1.8560056686401367, -0.22437359392642975, -0.5258949398994446, 0.5594707727432251, -0.8856609463691711, 3.084646463394165, -1.0640757083892822, -0.8238375186920166, -0.919736385345459, 0.8635029792785645, -0.47783029079437256, -0.5092834830284119, -0.1574515700340271, -0.4317828416824341, 0.14955173432826996, -0.6784935593605042, -1.4703985452651978, 0.30740687251091003, -1.8655081987380981, 1.0106806755065918, 0.31959110498428345, 1.1654012203216553, 1.3635834455490112, -0.9415153861045837, -0.5966653823852539, 1.1601096391677856, -1.7819430828094482, -0.16004320979118347, -0.11814378201961517, 0.3382108509540558, -1.3611711263656616, -0.7027940154075623, -1.5273237228393555, -0.8559970855712891, -0.06422016769647598, 0.07045672088861465, -0.14350353181362152, -0.6138055324554443, -0.8771029114723206, 0.047209251672029495, 0.5591081380844116, -1.1025960445404053, 1.8518105745315552, 0.11820748448371887, -0.58627849817276, -0.6306551098823547, 0.0193829033523798, 0.5478807687759399, 0.5614979267120361, 0.029314439743757248, -0.8598300218582153, 1.2017688751220703, 0.017083818092942238, -0.29777035117149353, 1.3861786127090454, 1.837063193321228, -0.4619734585285187, 0.19089113175868988, 0.9414905905723572, -1.299164891242981, 1.0919846296310425, 0.7496779561042786, -0.010179422795772552, 0.6495978236198425, -0.09066270291805267, -2.159205198287964, -0.9268139600753784, 0.6956284642219543, -0.012958302162587643, -1.2371680736541748, -0.35600289702415466, 0.7344687581062317, -0.775566041469574, 1.0190455913543701, 0.7411176562309265, 0.3410598337650299, 0.813362717628479, -0.9658582806587219, 0.17524228990077972, -0.5927488207817078, -1.5006242990493774, 0.44495484232902527, 1.2448594570159912, -0.9267497062683105, -1.2313987016677856, 0.7491569519042969, 1.086682915687561, -0.4252524971961975, 1.4506181478500366, 1.5184916257858276, -1.9673497676849365, -0.9971127510070801, 1.326066493988037, -0.5971192121505737, -0.3467038869857788, -0.020257167518138885, -0.2748483121395111, -1.1722760200500488, 1.0485979318618774, -0.11878462135791779, 0.44391918182373047, 1.5311646461486816, -0.37757083773612976, 0.42491868138313293, -0.5852200388908386, 0.14710624516010284, 0.2837800979614258, -2.150893211364746, 0.6854469180107117, 0.9505962133407593, 0.7969160676002502, 0.281024694442749, -1.188783884048462, 0.18383167684078217, 0.5382936596870422, -0.26172658801078796, -0.8699862957000732, 0.03097275272011757, -0.8113969564437866, -0.6061572432518005, 1.6818501949310303, -2.2671287059783936, -0.5701040625572205, -0.49323412775993347, -2.0235273838043213, -0.007568285800516605, 1.3075040578842163, 0.8150373697280884, -0.378537118434906, -1.165642499923706, -0.8946046829223633, 0.915195643901825, 1.0388678312301636, 0.048943616449832916, 0.8515879511833191, -1.1329948902130127, -0.10273633897304535, 0.9538130760192871, 0.35015183687210083, -0.3004085123538971, 1.2473818063735962, 1.270729660987854, -2.284214973449707, -0.46363765001296997, 1.4420431852340698, 0.7537528276443481, 1.368523120880127, 0.11144296824932098, 0.3988091051578522, -0.6392759084701538, 0.13295066356658936, 0.6148043870925903, -0.8959694504737854, -0.9744212627410889, -1.2070717811584473, 0.455527126789093, 0.6147217154502869, -1.0294251441955566, 1.160995602607727, 0.06821942329406738, -1.4277712106704712, 0.4892619550228119, 1.2067527770996094, -0.35013264417648315, 1.1460843086242676, 0.548748254776001, 1.5961905717849731, 1.154118299484253, 0.5095701217651367, -0.07184401899576187, 1.0822116136550903, -0.4653591811656952, -0.5298620462417603, -1.1922736167907715, 0.8825876712799072, -1.7694274187088013, 1.1722743511199951, -2.080310106277466, -0.9759420156478882, -1.117720365524292, 0.14937803149223328, -1.4429608583450317, -0.01563430391252041, -1.2850878238677979, 0.3093147873878479, 0.6343069076538086, -1.1464887857437134, -0.47607457637786865, 0.34084784984588623, -0.40216919779777527, 0.5199253559112549, -1.4774154424667358, 1.2946830987930298, -0.5080097913742065, -0.6568661332130432, -1.2212897539138794, 0.25711163878440857, -0.08846446871757507, 0.5742314457893372, 0.2882190942764282, 0.2839752435684204, -1.4326740503311157, -0.8894423246383667, 0.7412011623382568, 0.6975820660591125, -0.09359437227249146, -0.8315865993499756, -1.2695857286453247, -1.4815707206726074, -0.06540385633707047, 1.3441658020019531, 0.06789802759885788, -0.31638792157173157, -0.10123567283153534, -0.46090173721313477, 0.6497552990913391, 0.7801120281219482, -0.5364017486572266, 2.8599629402160645, 0.10508638620376587, 0.06745877116918564, -0.7485043406486511, 1.1692768335342407, 0.3849138915538788, -0.6088376045227051, -0.2048639953136444, 1.0699336528778076, -1.908221960067749, 1.2572687864303589, 0.43551942706108093, -0.8499947190284729, -1.8607866764068604, 0.7138776183128357, -1.403521180152893, 0.015181479044258595, -0.2223062664270401, 0.7893797159194946, 0.2464161515235901, 0.032408908009529114, 0.48668015003204346, 1.5306212902069092, -0.26611968874931335, 0.4500775635242462, -0.06254824995994568, -1.5751510858535767, 0.5321662425994873, -1.0480635166168213, -1.268546462059021, -1.1159220933914185, 1.6364710330963135, 1.0263946056365967, -0.398074209690094, -1.0263440608978271, -1.4965310096740723, -0.14756855368614197, -0.8910863399505615, 0.4003511071205139, 0.9794723391532898, 1.263153076171875, -1.211475133895874, 0.8394893407821655, -0.9074456691741943, -0.8626623749732971, -0.27675673365592957, -0.43337076902389526, 0.7368846535682678, 0.526846706867218, 1.5722428560256958, 0.14907531440258026, 0.992428183555603, -0.6590901017189026, 1.640811800956726, -0.8919944167137146, -0.7320170998573303, -1.1066334247589111, 1.6289739608764648, -0.7635530829429626, -0.11774443089962006, -0.7404131889343262, 0.02106480672955513, -1.2202911376953125, 0.10173138976097107, 0.30678480863571167, 1.7002826929092407, 0.8932514190673828, -0.7537760138511658, -0.9719266891479492, -1.3198031187057495, -0.07060755044221878, -0.7173315286636353, -1.395788311958313, -1.8513597249984741, 0.6178365349769592, 1.4275318384170532, 1.177642583847046, -1.4988853931427002, -0.4959982931613922, 0.3576614558696747, -0.5604247450828552, -1.526251196861267, -1.2953031063079834, 1.1204922199249268, -0.9708876013755798, -0.02513168193399906, -0.48209482431411743, -0.9166572093963623, 0.04417163133621216, 0.712311327457428, -0.9720004200935364, -0.24489495158195496, -0.3882567286491394, 1.0193464756011963, 0.029880570247769356, -1.566582202911377, -0.8651622533798218, -0.4309893548488617, 1.4801067113876343, -0.9924131035804749, 0.5968056917190552, 0.05925549566745758, -0.8656046390533447, 0.4615054726600647, 1.3523973226547241, -0.37255439162254333, 0.3611217141151428, -0.9854997992515564, -0.24130208790302277, 1.3868408203125, 0.3291398882865906, 0.11517776548862457, 1.381593942642212, -0.7688442468643188, 0.19245916604995728, 0.6129166483879089, 1.1840755939483643, -0.32262861728668213, -0.12434465438127518, 1.6428121328353882, -1.0258903503417969, -0.3171800673007965, 0.1126973107457161, 0.18237677216529846, -0.7995702624320984, 1.105989694595337, 0.00998762622475624, 0.18041114509105682, 1.4637664556503296, 1.4035508632659912, 0.8227208256721497, 0.9040852785110474, 2.9576404094696045, 1.1075624227523804, -0.4813080430030823, -0.7194573879241943, -0.8919242024421692, -2.0495893955230713, 0.8294411897659302, -0.6389020085334778, 1.8633670806884766, -0.18491733074188232, 0.585809051990509, 2.363579750061035, 0.7341932058334351, -0.8007022142410278, -1.5915451049804688, -0.32698193192481995, -0.4486667215824127, 0.32237520813941956, -0.0064767757430672646, -1.9410218000411987, -0.44487321376800537, -0.8303346633911133, -0.5899572372436523, -0.821876585483551, -0.8833673000335693, 0.032731425017118454, -0.957970142364502, -1.1936124563217163, 0.6881375908851624, -0.6565336585044861, -0.45305880904197693, 1.311975121498108, 0.5328494310379028, 1.3246959447860718, -0.829612135887146, -1.0862282514572144, -0.5752453804016113, 0.17953336238861084, -0.4955666959285736, -1.8761980533599854, -0.41369789838790894, 0.2183685302734375, -0.7954877614974976, 1.090126872062683, 0.7592087984085083, 0.06866152584552765, -0.7560296058654785, -1.3662943840026855, -0.17194095253944397, -1.1645303964614868, -0.5806599855422974, 1.4508349895477295, -1.4111042022705078, 0.6313235759735107, 1.348578929901123, -1.606117844581604, 0.6959260702133179, 0.9251189827919006, -0.9227353930473328, 1.342740535736084, 0.5661807656288147, -0.49476829171180725, 1.7365577220916748, 0.26136791706085205, 1.1202282905578613, -0.2282055765390396, 0.5677645206451416, 1.0276449918746948, 0.42515426874160767, -0.0030480637215077877, -0.9495301842689514, 0.020661797374486923, 0.2083931714296341, 0.22650769352912903, 1.6784619092941284, -1.5058174133300781, 0.039985015988349915, 1.9311424493789673, 1.6957467794418335, -1.2383778095245361, -0.32670605182647705, -0.7205557823181152, -1.5032926797866821, -1.4461036920547485, -0.8331480622291565, 0.23989678919315338, -1.8938343524932861, 1.2324575185775757, -0.6942697167396545, 0.3410382568836212, 1.932364583015442, -0.5845600366592407, -1.1656900644302368, -1.6102181673049927, 1.930446743965149, -0.30417004227638245, 0.7258777618408203, -0.46592262387275696, 0.32740503549575806, -0.8699591159820557, -0.2713511884212494, -0.7447600364685059, -0.3378017544746399, -0.7810521721839905, 0.09463447332382202, 1.2210805416107178, 0.8142836093902588, 2.1681461334228516, 0.2388635277748108, -1.9762872457504272, 1.4756836891174316, 0.6942840814590454, -0.9676187038421631, -0.014279579743742943, 0.9743165969848633, 2.002328634262085, -0.583928108215332, -0.8646663427352905, -1.3389368057250977, 0.33027568459510803, -0.5175966024398804, 0.23661848902702332, -0.5029343962669373, -0.02118440344929695, -1.6811543703079224, -1.3333390951156616, -1.2902253866195679, -0.3929642140865326, 2.570749044418335, -1.6336591243743896, 1.7394073009490967, 0.0006425062310881913, 1.5919413566589355, 1.1111990213394165, 0.27682992815971375, 0.5067181587219238, -0.8335698246955872, -1.4622308015823364, -0.8541417121887207, 1.9282996654510498, 0.37972429394721985, -1.4331709146499634, -0.9680251479148865, 1.6991853713989258, 1.8850152492523193, 0.013029258698225021, 0.11819857358932495, -1.0553574562072754, 0.5472376346588135, 0.1454745978116989, 0.039538491517305374, -0.28069114685058594, -1.257320761680603, 0.6964598298072815, -0.10491462796926498, -0.23395614326000214, -1.6429771184921265, 0.4085608422756195, 0.39006829261779785, -0.7331656813621521, 1.3587312698364258, -0.19484373927116394, 0.2117961049079895, -0.6287140846252441, 1.7443885803222656, 0.21415473520755768, 0.7269644141197205, 1.8151042461395264, -0.05399075523018837, -0.9773785471916199, -1.540437936782837, -0.8722285032272339, -1.9308112859725952, -0.6761055588722229, 0.7441504597663879, -1.508023977279663, 0.8265978693962097, -1.4463740587234497, -0.15512481331825256, 1.0947715044021606, -0.04240138456225395, 0.238884836435318, -0.9678716659545898, -0.2527967691421509, 0.446313738822937, 0.14156563580036163, -1.2593997716903687, 0.32056525349617004, -0.11262395977973938, -1.3083332777023315, 0.6495218873023987, 0.15293742716312408, 0.11816547811031342, 1.4947526454925537, -0.37029460072517395, 2.008114814758301, 0.27158135175704956, 0.048063214868307114, 0.7206501960754395, -0.04177961125969887, -0.5622061491012573, -0.027535583823919296, -0.17321933805942535, 0.7181575298309326, 1.5261377096176147, -0.26389849185943604, -1.0030407905578613, -0.04848689213395119, -0.21872003376483917, -1.7048799991607666, 1.0244457721710205, 0.7932477593421936, -1.2068142890930176, 0.09200077503919601, -0.6885411143302917, 0.28114885091781616, 0.6752942204475403, -1.8073025941848755, -1.5777333974838257, 0.5840768814086914, 0.7052242159843445, -0.9002032279968262, 0.1946643441915512, -0.009191776625812054, 0.6515457630157471, -0.08330726623535156, -0.7358261346817017, 0.7839617133140564, 0.17356540262699127, 1.1026849746704102, -0.40891727805137634, 0.11959695816040039, -0.9374182224273682, 1.2328100204467773, -1.8940051794052124, 0.0819941908121109, -1.6866334676742554, 0.5776201486587524, 0.5533458590507507, -0.669486403465271, -0.002536401152610779, 0.037496600300073624, -0.41716814041137695, 0.1115884855389595, -1.0914297103881836, -2.108227014541626, -0.3210350275039673, -0.8513522148132324, 1.745098352432251, -0.3739751875400543, 0.20898810029029846, 0.17869932949543, -1.7821370363235474, -0.39735397696495056, -0.02938893251121044, 0.5542208552360535, 2.238145112991333, 0.692937970161438, -0.05372597277164459, 1.0948673486709595, -0.9216350317001343, 0.4703610837459564, 0.1457081139087677, -0.4243374466896057, -0.1279841512441635, 1.7382317781448364, -0.37646743655204773, 0.20454804599285126, -0.0052711437456309795, 0.5775604844093323, 0.5323547720909119, -1.1322814226150513, 0.4354526698589325, 0.3437919020652771, -0.8870677947998047, -0.6187957525253296, 0.5875118970870972, -0.7454352974891663, -0.06852440536022186, 0.5441735982894897, -0.3162315785884857, 1.818217158317566, -0.017841622233390808, -0.0860663652420044, 0.7277683615684509, 1.3719745874404907, -0.04998583719134331, 0.9166210889816284, -0.5115281939506531, 0.5717926621437073, -1.4149962663650513, 0.35602012276649475, 0.35819482803344727, -0.0342533104121685, -0.38636311888694763, -0.09282573312520981, 0.26104164123535156, 0.8918042182922363, 1.8720051050186157, 0.2767341434955597, 0.7218112945556641, -0.9150763750076294, 0.43937936425209045, 0.06526295840740204, 0.6129964590072632, -0.4749143421649933, -1.1096752882003784, 1.815077304840088, 1.0900850296020508, 0.018573930487036705, -2.1090402603149414, -0.22261519730091095, 0.0006791273481212556, 0.18077029287815094, 0.5314986109733582, 0.7076907157897949, -0.25026628375053406, -0.8045916557312012, -1.8842271566390991, -0.43377313017845154, -0.8943247199058533, 0.35029202699661255, -0.9201849102973938, -0.20973743498325348, -1.8079476356506348, 0.5574826002120972, -1.4624193906784058, 0.5803983211517334, 0.585699737071991, 1.1947461366653442, -0.33271005749702454, 0.5533355474472046, 2.010843515396118, 0.14527031779289246, 1.0003536939620972, 0.31021007895469666, -0.10207345336675644, -1.7264783382415771, 1.5251320600509644, -0.41611534357070923, 1.8219094276428223, -1.4723972082138062, 0.781273365020752, 0.6420766711235046, 0.0902189314365387, 0.10073975473642349, -0.6163722276687622, -1.4545516967773438, 2.4646244049072266, 1.4052015542984009, -0.2659611701965332, -1.6370899677276611, 0.2536921501159668, 1.5920990705490112, -0.3597334325313568, 0.4381776452064514, -1.7592953443527222, -0.994330108165741, -0.725058913230896, -1.4986376762390137, 0.47566965222358704, -0.1327870786190033, 0.8968674540519714, 1.9753930568695068, -0.6826338171958923, -0.08358927071094513, 0.4981578290462494, -0.7643974423408508, -1.1728379726409912, 0.07348362356424332, 0.19196070730686188, -1.6956062316894531, 0.12770798802375793, -2.2287399768829346, -0.5839188098907471, 0.7081590294837952, 0.8415904641151428, -0.7735862731933594, 0.7370346188545227, -0.5470302104949951, 0.3954763412475586, -1.1324865818023682, 1.5716619491577148, 0.7694554924964905, -0.1972833126783371, -0.01634226180613041, -1.8078466653823853, 0.45335906744003296, -0.45582708716392517, -0.08402549475431442, 0.04648109897971153, 2.120666980743408, 1.109842300415039, -0.6010242104530334, 0.23951110243797302, 0.3804468512535095, -2.7814252376556396, -0.8011180758476257, 0.6475176811218262, 0.4931000769138336, 0.9993235468864441, -1.2604527473449707, -0.5451884865760803, 0.5231437087059021, 0.2639492154121399, -1.2105263471603394, -0.7405393719673157, -0.09492752701044083, 0.6534978747367859, 0.3471531271934509, 1.5782825946807861, -0.0945083498954773, 0.29517728090286255, -1.2769262790679932, 0.9232704043388367, -0.18332765996456146, 0.04653006047010422, 0.7682851552963257, 0.545989453792572, 0.7154923677444458, -1.2931429147720337, 1.4640978574752808, -0.7219284176826477, 1.033075213432312, 0.7666181921958923, 0.7390316128730774, 0.6044892072677612, 0.3623349070549011, -0.19642722606658936, -0.25965696573257446, -0.3969550132751465, -0.6633504033088684, -0.3451343774795532, -1.627233862876892, 0.23733393847942352, 0.9929364919662476, -0.7839997410774231, -0.9215231537818909, -1.3040024042129517, -1.8463186025619507, 0.5836753249168396, -0.48698627948760986, 0.04695913940668106, -1.759283423423767, 1.7901496887207031, 0.6652159094810486, 0.9054563045501709, -1.3632848262786865, -0.8115912079811096, 0.3560025990009308, -0.5002995133399963, 0.7239171862602234, 0.5757662057876587, 1.0058141946792603, -0.22663503885269165, 0.870149552822113, -0.9265767931938171, 1.5705050230026245, 2.287517786026001, 1.018664836883545, -0.5306094884872437, 0.12343884259462357, -0.2952793538570404, 0.3989964425563812, 0.24353602528572083, 1.6092467308044434, -1.1651194095611572, 0.5150609612464905, 0.21992617845535278, 0.9552311897277832, -1.5211632251739502, 1.7818299531936646, -2.4724478721618652, 0.03340267017483711, -0.8383764028549194, 0.7351166605949402, -0.7015381455421448, 0.16077280044555664, -1.3064557313919067, -0.044731203466653824, 0.5554159283638, 0.38914576172828674, -0.28324446082115173, 1.795637607574463, -1.6533737182617188, -0.6858395338058472, 0.274723619222641, -0.2342483550310135, -1.5352150201797485, -0.34629106521606445, -1.6248036623001099, 0.411359578371048, -0.7680161595344543, -0.39106285572052, -1.721454381942749, -1.1650336980819702, -0.5992630124092102, 0.8728578090667725, -0.8085802793502808, -1.4205620288848877, -0.5376473069190979, 1.352899193763733, 1.5570753812789917, -0.38770464062690735, 1.284096360206604, 1.1534632444381714, -1.1390178203582764, 0.6424070000648499, 1.1494591236114502, 1.4561882019042969, 0.2643531262874603, -0.0701005756855011, 0.2955300509929657, -0.11994252353906631, 1.0531120300292969, -0.3732447922229767, -0.026372205466032028, 0.3451131582260132, -0.916111946105957, -0.5595105290412903, 0.21574798226356506, 0.13646076619625092, -0.852252721786499, -1.9297380447387695, -1.4468867778778076, 0.26029661297798157, -0.1264246553182602, -0.6168134808540344, 0.06209571287035942, -1.7315735816955566, -0.12674884498119354, 0.7234352231025696, 0.4094787836074829, 0.38938504457473755, 0.05602944269776344, -0.1453092098236084, -0.35305148363113403, -0.5484811067581177, 0.39567866921424866, -1.793875813484192, -1.7818366289138794, -0.09702549874782562, -0.7676879167556763, 1.9420045614242554, 1.1627014875411987, 0.21763378381729126, 0.060925256460905075, -1.7534334659576416, -1.5915822982788086, -0.8936445116996765, 0.7708600759506226, 0.1968446671962738, 0.7193440198898315, 0.06811609864234924, 0.7763517498970032, 0.9755931496620178, -0.21598581969738007, 0.6376509070396423, -1.883978247642517, 0.872162938117981, -0.8924828767776489, -0.8019199967384338, -0.11103237420320511, -0.6840009093284607, 0.3547770380973816, -1.4274871349334717, 0.9079747796058655, 0.22760009765625, -0.06146205589175224, -0.03060694970190525, -0.9798767566680908, 0.16059161722660065, -0.21795684099197388, 1.1339772939682007, 0.6772281527519226, 0.9451150298118591, 1.035826325416565, -0.0008701533079147339, 0.2720678150653839, -1.6760327816009521, 0.43850940465927124, 1.2991950511932373, 1.3670780658721924, 0.355234831571579, 0.09262093901634216, 1.5299543142318726, 0.14060044288635254, -1.390775203704834, 0.05693463608622551, -1.1768693923950195, -0.4155004620552063, 0.704460084438324, -0.6010005474090576, 0.174397274851799, 0.6402075290679932, 0.8531313538551331, -0.2352367788553238, -0.35565420985221863, 0.8712440133094788, 1.6902308464050293, -1.2269421815872192, -1.6491801738739014, 0.2599267065525055, -0.07695040851831436, 0.8878954648971558, -0.8574252724647522, -0.6851561069488525, 1.4118998050689697, -1.3721752166748047, -0.8366134166717529, 0.7194199562072754, 0.7535651922225952, -0.4970264732837677, -0.5888087153434753, -1.4315158128738403, 0.1607116162776947, -0.6791881918907166, 0.47149524092674255, 0.18249128758907318, -2.0727906227111816, -0.3241727650165558, -1.993770956993103, 1.4756075143814087, 0.07791386544704437, 0.20294393599033356, -0.5267860293388367, -0.03790314868092537, 0.639726996421814, -0.6496464610099792, 1.076562523841858, 0.5263563990592957, 1.8004120588302612, 0.35245341062545776, 1.50009024143219, 1.6698521375656128, 0.8023168444633484, 0.5128716826438904, 1.4539536237716675, 0.9149024486541748, -0.9472006559371948, -0.02533149905502796, -0.4326372742652893, 0.35727739334106445, 0.6799312829971313, 0.47504645586013794, 1.4705336093902588, 1.210198998451233, -0.6520727872848511, -1.4349277019500732, 0.43579307198524475, -1.9591286182403564, -0.4062960743904114, -0.44068634510040283, 0.2228459119796753, 0.06332462280988693, -0.07720060646533966, 2.9767072200775146, -0.5851355791091919, -0.290188729763031, 1.350714087486267, 0.46601542830467224, 0.3114314377307892, -0.0422569140791893, -1.4024178981781006, 0.053019873797893524, 0.41416117548942566, 1.163525938987732, 1.3560888767242432, 0.23936299979686737, -0.6545792818069458, -0.9235248565673828, 0.7059462070465088, 1.189387321472168, -0.0008880148525349796, 0.33637967705726624, 0.6714540123939514, -0.2825442850589752, 0.24913690984249115, 0.6777125597000122, 0.10432203114032745, 0.07974760979413986, -0.31061092019081116, -0.7376080751419067, -0.5475263595581055, 0.9521651268005371, 1.0627766847610474, -1.435872197151184, -0.730137288570404, -1.1451282501220703, 2.167616844177246, -2.2246315479278564, -0.41818106174468994, 0.5523219108581543, -0.13253897428512573, 0.3016347289085388, -0.5966551899909973, -1.2477504014968872, 1.4849120378494263, -0.3281608819961548, -1.7900034189224243, 0.7824485898017883, -1.056251883506775, 2.715009927749634, -0.6738789081573486, -0.47631707787513733, 1.3843215703964233, -1.740935206413269, -1.2782936096191406, 0.468110591173172, 1.1271766424179077, -0.8653950691223145, -0.38502606749534607, 1.1504013538360596, 0.16126544773578644, 1.6019443273544312, -0.6559335589408875, -2.2287213802337646, 0.17883528769016266, -0.5072218775749207, -0.5696470737457275, -0.013361281715333462, -0.32818204164505005, -1.479091763496399, 0.18003928661346436, -0.9057676196098328, 0.2867017388343811, 1.7032240629196167, 0.048141106963157654, -0.773095428943634, -0.6384793519973755, 0.6617262959480286, -0.4596630036830902, 0.7767514586448669, -1.3010940551757812, 0.26629775762557983, -1.099075436592102, 0.37281253933906555, 1.5965944528579712, -0.8419760465621948, -1.2307199239730835, -2.2409181594848633, -1.7995285987854004, 0.8044866919517517, 1.7904188632965088, 0.2963297367095947, -0.20097769796848297, 0.3913481533527374, -1.8949881792068481, -1.295061469078064, 1.2153784036636353, 1.670656681060791, 1.2344993352890015, 1.1001204252243042, -0.31956207752227783, -0.2926246225833893, 0.913890540599823, 0.175750732421875, -1.4128053188323975, -1.2682573795318604, -0.43554845452308655, -0.4655711352825165, -2.461264133453369, -0.9540699124336243, -1.0862178802490234, 2.354057788848877, 0.7911161780357361, -0.18911488354206085, 0.5295031070709229, 0.7022464871406555, -1.063346028327942, -0.2902396619319916, 0.9408233761787415, -0.33824360370635986, 0.9886799454689026, -0.39150092005729675, -1.4977378845214844, -1.201197624206543, 0.37908604741096497, -0.8010980486869812, -1.8342268466949463, 1.1680524349212646, 0.49029019474983215, -0.29485899209976196, 0.10461270809173584, 0.16237866878509521, 1.059125304222107, -0.1852564960718155, -0.32558220624923706, -0.6059207320213318, -1.3385778665542603, 1.168319821357727, 0.5136449933052063, -1.0347868204116821, 0.08893007785081863, 1.8261617422103882, 0.02605321817100048, -1.8361042737960815, -0.8763760328292847, 0.32399782538414, -0.6092226505279541, 2.5661091804504395, -0.8923203349113464, 1.0909136533737183, -1.4964535236358643, -0.028698071837425232, -1.917396903038025, 0.19500231742858887, 0.8802657723426819, 2.286655902862549, -0.5637630224227905, -0.860026478767395, 2.290419578552246, -1.3366048336029053, 0.6060933470726013, -0.45804670453071594, 0.6944952011108398, -0.40019795298576355, -0.7259854674339294, 1.6470824480056763, 0.9808973073959351, 0.35987141728401184, 0.634909451007843, 1.159855842590332, -1.7151564359664917, -0.4864749610424042, 2.0358119010925293, 0.4449416697025299, 0.23660361766815186, 0.9641693830490112, -0.8949785828590393, -0.021622074767947197, -0.6250436305999756, -1.4970968961715698, 0.023246901109814644, -0.1862165480852127, -0.0836542621254921, -0.8594025373458862, -0.35475674271583557, 0.2104513794183731, 0.3905942142009735, -0.10405085980892181, -0.7770746350288391, -2.0903213024139404, 1.2172858715057373, 1.126493215560913, 1.263810157775879, 1.9913673400878906, -0.6460287570953369, 0.32505732774734497, 0.5139806270599365, 1.092822790145874, 0.8573914766311646, -2.082169771194458, 2.5408895015716553, -1.9622678756713867, -1.105880856513977, -1.093604564666748, -1.3543922901153564, 1.2798691987991333, -1.3203800916671753, 0.833282470703125, 1.2961102724075317, -0.8033930659294128, 0.6834202408790588, 0.09541796892881393, -0.3566247820854187, 0.5684022903442383, 1.9051287174224854, 1.628310203552246, 0.777316689491272, 1.5158122777938843, -1.045634388923645, -0.7125810980796814, -0.1372985541820526, 1.4785218238830566, 1.0346379280090332, -0.6090080738067627, 0.9513916373252869, -0.7098460793495178, -0.14820507168769836, -0.8925225734710693, 1.3846189975738525, 0.5434086918830872, 1.3264577388763428, 1.8468644618988037, -0.8015012741088867, -0.5072871446609497, 0.482855886220932, 1.6346155405044556, 0.24837641417980194, 0.31863921880722046, 1.6905169486999512, 0.4909053146839142, -0.4516226649284363, -1.4856975078582764, -0.6821410655975342, 1.5835415124893188, -0.9372066855430603, 0.027705803513526917, -0.4190330505371094, -0.9827384352684021, -0.774921715259552, -0.42755913734436035, -0.7206875681877136, -0.09204244613647461, -0.36505186557769775, -0.18063661456108093, -1.0848760604858398, 0.6003469824790955, 1.3126674890518188, 1.247290015220642, 0.4778834879398346, 0.5856991410255432, -0.18018269538879395, 0.44917574524879456, 0.39473557472229004, 2.0602426528930664, 0.9969490766525269, 1.897719383239746, 0.08232580870389938, 0.6948878169059753, -0.23620052635669708, 0.7486191987991333, -0.8879519104957581, -0.28428560495376587, -0.3761943578720093, 1.2402175664901733, 0.9097046852111816, -0.8083577156066895, -1.5546671152114868, -1.4451653957366943, -1.5479750633239746, -1.0493708848953247, -0.05739971995353699, -2.1648457050323486, -0.7947807908058167, 0.5466724634170532, -0.3476933240890503, 0.4830033481121063, -1.2420518398284912, -0.22694526612758636, -1.2692341804504395, 0.622701108455658, -0.12353511899709702, 0.9662380814552307, 2.120690107345581, 0.5718235969543457, -0.09960618615150452, -0.5364685654640198, 0.5887389779090881, -0.21363204717636108, 0.30687063932418823, -0.12352283298969269, -1.0416336059570312, 1.2725752592086792, -0.5805290937423706, 1.737317442893982, -0.6497963666915894, -1.950724720954895, -2.2817208766937256, -0.32264187932014465, 0.9248577952384949, -0.567298173904419, 0.6543440222740173, 0.8613268733024597, -0.16385002434253693, -0.6934324502944946, -0.9912139773368835, 1.0160720348358154, -0.08129566162824631, 1.5215682983398438, -0.6269931197166443, -1.1550288200378418, -0.008503246121108532, -1.5431057214736938, -0.8341385126113892, 0.7008342742919922, -1.4243223667144775, -0.6582763195037842, 0.6181714534759521, -0.3481481969356537, -0.28352558612823486, 0.5649526119232178, 0.9816697835922241, -1.7644944190979004, -1.9819087982177734, -0.1722264587879181, 0.9749840497970581, -0.0592336431145668, 1.7277532815933228, -1.221856951713562, -0.8501419425010681, 0.7004331946372986, -1.6129560470581055, -1.531926155090332, -0.08689969778060913, 1.2606379985809326, 1.2954254150390625, -0.5129035115242004, -1.4835411310195923, 1.2021758556365967, -0.5181654095649719, -0.24392879009246826, -0.4374927878379822, -0.7660645246505737, 0.1923568844795227, 1.407732367515564, 0.2779451608657837, 0.9674700498580933, 0.30206888914108276, -1.5478419065475464, -1.321216344833374, -1.1165639162063599, -0.6174235343933105, -1.396082878112793, -1.1354255676269531, 0.5258146524429321, 0.4736382067203522, 2.1303858757019043, -8.592647645855322e-05, 0.09621381759643555, -0.9206749200820923, -0.29720795154571533, 1.2495509386062622, -1.3327146768569946, -0.025577036663889885, -1.1830387115478516, -0.14218497276306152, 2.703953504562378, 0.8653303980827332, -1.2428325414657593, -3.530674934387207, 0.7871127724647522, -2.0336406230926514, -0.2174457162618637, -1.4143022298812866, 0.8015451431274414, -0.3278248608112335, -0.6828787326812744, 0.6756470799446106, 0.9050363302230835, 0.24029965698719025, -1.6560280323028564, -0.6280542612075806, 1.032069444656372, -2.032174825668335, -2.554368257522583, 1.1877694129943848, -0.24979421496391296, 0.8272569179534912, 0.8247644305229187, -0.12557953596115112, -0.6605157256126404, -0.40657639503479004, 0.504544734954834, 1.0560497045516968, 0.971483051776886, -0.26958167552948, 0.4052882790565491, 0.13938476145267487, 0.3983602523803711, 0.7721923589706421, 0.172966867685318, -2.2213528156280518, 1.0340726375579834, -1.9047489166259766, -0.34720271825790405, -0.8117367625236511, -0.8651323318481445, 0.25481829047203064, -0.27195408940315247, 0.2856077551841736, 1.185281753540039, 1.8842365741729736, 0.841802179813385, -0.4425356686115265, -1.1001813411712646, 0.030889147892594337, -0.24528585374355316, 1.7895464897155762, -0.04475386068224907, -0.47572433948516846, 0.7109814286231995, 0.5187169909477234, 0.523276150226593, 0.214812770485878, -0.085037961602211, -0.25106939673423767, -0.6110848188400269, -0.7887175679206848, 1.3310303688049316, -0.9404072165489197, 1.6125375032424927, 1.5788854360580444, 0.44927671551704407, 2.190289258956909, 0.6982913613319397, 0.66921067237854, 0.28603947162628174, -0.8470760583877563, -0.5645437240600586, 1.2753812074661255, -0.0947272852063179, 0.042858682572841644, 1.3799954652786255, -1.1651625633239746, 0.4148823320865631, -0.2427898347377777, -1.7691599130630493, -0.4121306538581848, -0.7446964979171753, 0.5572453141212463, 1.4937561750411987, -0.4605744481086731, 0.6724761724472046, -0.2837729752063751, -1.8798903226852417, -0.5959212183952332, -0.45180264115333557, 0.04505826532840729, -1.1848684549331665, 0.13150593638420105, 1.431738018989563, -0.04018506035208702, -0.7451223731040955, -0.44001927971839905, -0.3446762263774872, 1.3133249282836914, -2.0763463973999023, 0.9258373379707336, -1.7236051559448242, 0.10541445016860962, 0.775031566619873, 0.4944700598716736, 0.22819599509239197, -2.3128645420074463, 0.9645535349845886, -0.16320963203907013, -1.2443712949752808, 0.4039890766143799, 0.3200203776359558, -1.770538091659546, -0.3273465633392334, 0.7778356671333313, -0.6189680695533752, 1.2085115909576416, -0.30282098054885864, -0.6549845933914185, 1.556477427482605, -0.7706227898597717, -0.9951931834220886, 1.5920151472091675, -1.4915082454681396, 0.5687689185142517, 0.12980413436889648, 0.3606143593788147, -0.21078439056873322, 0.6072463393211365, 0.3021511137485504, -1.9390246868133545, 1.0484991073608398, 1.942674994468689, -0.5366895794868469, -1.2434892654418945, 1.3834290504455566, 2.853494644165039, -0.19516658782958984, 1.7951256036758423, -1.5975964069366455, -0.7626796364784241, 0.11851431429386139, 0.4135276973247528, -0.3286178708076477, 1.5460203886032104, -1.089913010597229, -1.5859414339065552, 0.5719178915023804, -3.80580472946167, -0.4879050850868225, 2.1824076175689697, 0.6362631916999817, -0.29337096214294434, -0.3529582619667053, 0.33700308203697205, -1.839380145072937, 0.13288739323616028, -0.4982009828090668, -0.8148212432861328, 1.349876880645752, 1.5060863494873047, 1.251549482345581, 0.7509818077087402, -0.42463064193725586, 1.2661826610565186, -0.5066109299659729, 0.8077986836433411, 0.44528862833976746, 0.2769685983657837, 0.49688467383384705, -0.32791054248809814, -0.5157872438430786, 0.4538787305355072, 3.45133638381958, -1.9209458827972412, 0.2332136332988739, -0.4189273416996002, 1.9516210556030273, -0.1905197650194168, 0.1843816488981247, -0.5588135719299316, -0.7804850935935974, 0.054800596088171005, 0.6071887612342834, 2.5575358867645264, -1.1104167699813843, 0.02601362206041813, 0.33954381942749023, -2.1358072757720947, -0.5487971305847168, 0.5665295124053955, -0.9484941363334656, -0.03839820250868797, -1.6920661926269531, 0.2665756642818451, 0.6487910747528076, 1.4368919134140015, 0.6639547348022461, 0.548850953578949, -0.9885948300361633, 0.5211122632026672, -0.604087769985199, 0.40809932351112366, -0.5244901776313782, 1.5351226329803467, -0.4865182042121887, 0.3521769642829895, 0.792037844657898, -0.1883811354637146, 0.7228943705558777, 0.9232257604598999, -0.35207125544548035, 1.2503231763839722, 0.7254006266593933, 0.3977735936641693, 1.7172911167144775, -1.0757646560668945, -1.6008752584457397, 1.0920180082321167, -0.0895865336060524, -1.2603412866592407, -0.6804468631744385, -1.0599290132522583, -1.2259001731872559, -0.667400598526001, 1.354780673980713, 1.4600337743759155, 1.0158876180648804, -0.4126964807510376, -0.19288212060928345, -0.5791076421737671, 0.28790080547332764, -1.0921764373779297, 0.34338197112083435, -1.3537729978561401, 2.0307133197784424, 1.9258885383605957, 0.05696342512965202, -0.30234360694885254, 0.11478956043720245, 1.8192775249481201, -0.6849105358123779, -0.5205072164535522, 0.2815721929073334, 0.5433743000030518, -0.2827252745628357, -0.1364886313676834, 0.8838830590248108, -0.8255487084388733, 1.0664035081863403, 1.2824610471725464, 1.5375796556472778, -1.3872569799423218, 0.0048998030833899975, -0.7902052998542786, 0.7611229419708252, -0.1402142196893692, 0.7224688529968262, 0.5742761492729187, 1.004483938217163, -1.7206014394760132, -0.7245892286300659, 2.0183568000793457, 0.5476002097129822, -1.2963701486587524, -0.9165894389152527, -1.3325140476226807, 0.9779730439186096, 0.4701598584651947, -0.05937841907143593, -1.886974811553955, -1.2481380701065063], [-0.23313327133655548, 0.011876881122589111, -0.07068034261465073, 0.9186455011367798, -0.6151625514030457, -1.5984545946121216, 0.620190441608429, 0.6201083064079285, -0.37027981877326965, 1.2875369787216187, 1.3761883974075317, 1.5121361017227173, 0.5489659309387207, 0.21966037154197693, 0.2235284298658371, 0.45201361179351807, -0.06323233991861343, 0.8173366785049438, -0.6859645247459412, 0.13918675482273102, -0.25825679302215576, 0.542990505695343, -0.3989700675010681, -2.2324306964874268, 0.9393404126167297, 1.1731536388397217, -0.02682962641119957, -0.9186043739318848, 0.1035841852426529, 0.8277150392532349, -1.0852391719818115, -1.1788723468780518, 1.222862958908081, 0.5545908808708191, 0.2452298253774643, -0.8722712993621826, -0.17501375079154968, 0.3899247944355011, -0.1645936220884323, 0.22532771527767181, -1.1076610088348389, -0.6555653214454651, 0.4062166213989258, 0.6396507024765015, -0.002457135124132037, 0.2708699703216553, 0.6157020926475525, -0.07098384201526642, -0.320061057806015, 0.9806526899337769, -0.3642941415309906, 0.6231070160865784, -0.12393157184123993, 1.5230376720428467, 0.22761081159114838, 0.7088860869407654, -0.2332647144794464, 0.3215799629688263, -0.2200237214565277, -0.6748070120811462, -0.07625125348567963, -0.8203681707382202, -0.9527690410614014, -0.25691673159599304, -0.5611295104026794, -0.32183489203453064, -0.14076606929302216, 0.5075134634971619, 1.1872014999389648, -0.3369399309158325, 0.17185866832733154, 0.727771520614624, 1.398176670074463, -0.33182111382484436, 0.4848310947418213, 0.8847615718841553, -0.3493902385234833, 1.1764910221099854, -2.214808702468872, -0.49905937910079956, -1.5817514657974243, -1.7812272310256958, 0.13371065258979797, -0.17113132774829865, 0.594581127166748, -0.08141879737377167, 1.6439173221588135, 2.2253549098968506, -1.7823023796081543, -0.6062046885490417, -0.09581490606069565, -0.37025031447410583, -0.6248225569725037, 0.5218995809555054, -0.579980731010437, -0.34958046674728394, 0.16866032779216766, 0.5889312028884888, -0.017684530466794968, 1.812971591949463, 1.1524256467819214, 0.18601590394973755, -0.008339165709912777, -0.2510960102081299, -0.5001533031463623, 1.489441990852356, -1.4196929931640625, -0.35200682282447815, -1.7223358154296875, 0.06029333546757698, 0.0908297598361969, -0.20191025733947754, -0.6855600476264954, -0.3163701295852661, -0.04271091893315315, 0.5145991444587708, 1.972924828529358, -0.36145687103271484, -1.2953215837478638, 1.0808435678482056, -0.9827097058296204, 0.002163682598620653, -0.16895133256912231, 0.7857654690742493, -0.8555324673652649, -0.3317449688911438, 0.270988792181015, -1.7526787519454956, 0.8176233768463135, 1.041881799697876, 2.0465176105499268, -0.40918540954589844, -0.8616999387741089, 0.6815145611763, 0.2980089783668518, 0.24725502729415894, -0.3276353180408478, 0.27685222029685974, 0.6312452554702759, 2.342480421066284, 0.2681483030319214, 1.0809730291366577, 0.8654306530952454, 0.8593370318412781, 0.15044201910495758, 0.2976607084274292, 1.3140863180160522, 0.5732330083847046, 0.137738898396492, 1.2319892644882202, -0.31494301557540894, 0.8284561634063721, 0.2686193883419037, -1.0412566661834717, -1.1987003087997437, 0.497104674577713, 0.19859249889850616, -0.1831265389919281, 0.05576855689287186, 1.7633599042892456, -1.293270230293274, 1.691210150718689, -0.33504730463027954, -1.4315487146377563, -0.35365673899650574, 0.18638506531715393, 0.7398566603660583, -0.6821608543395996, -1.593136191368103, -0.03500889614224434, 0.1438320279121399, 1.450863242149353, -0.979275643825531, 0.3619001507759094, 0.5829927325248718, -1.8445186614990234, -1.0844084024429321, 0.27218809723854065, -0.4060298204421997, 0.4534653425216675, 1.231865406036377, -0.1815125197172165, -1.1557703018188477, -1.6264488697052002, 0.3976946771144867, 1.4504382610321045, 1.9910451173782349, -0.5186271667480469, 0.1820070594549179, 0.21491487324237823, -0.2978098392486572, -1.1975164413452148, -0.17156878113746643, 0.23942112922668457, -0.22021272778511047, 0.16241782903671265, 0.05630607157945633, -0.4575631320476532, 1.642532229423523, -0.7220540642738342, -0.35078975558280945, -0.4661270081996918, 0.45291370153427124, -0.5349478125572205, 0.1935373842716217, 0.4828244149684906, 1.4742920398712158, 1.0882285833358765, -0.15280519425868988, -1.7994235754013062, -3.0298213958740234, -0.7478132247924805, 0.09805559366941452, -2.035757064819336, 0.10555189847946167, 0.01282463688403368, -0.44852596521377563, 0.814662516117096, -0.17511051893234253, -0.11304714530706406, -1.3967621326446533, -1.138352870941162, -0.509261965751648, 0.827504575252533, 0.5244652032852173, 0.9321315288543701, -0.2778175473213196, 2.076533317565918, 0.6763202548027039, 2.6449480056762695, 0.03168206289410591, -0.3852890133857727, -1.2395548820495605, 0.3285464644432068, -1.4620723724365234, -0.13421490788459778, 0.1746508777141571, -1.1948968172073364, -0.9564350247383118, -1.771925449371338, -0.7295148968696594, 0.05372583866119385, 1.2246965169906616, 0.7787209153175354, 0.9872470498085022, -1.5865824222564697, 0.20126739144325256, -1.0709495544433594, -0.07374069839715958, 0.29804515838623047, -0.42932945489883423, 0.4055456817150116, 0.9132986664772034, 0.6227635145187378, -1.1260188817977905, -1.1286706924438477, 1.17603600025177, 0.8955855965614319, 0.21004703640937805, -1.7838895320892334, -0.4336736500263214, 2.7397994995117188, 0.6695424914360046, -1.5502997636795044, -1.4723541736602783, -1.3139152526855469, -1.3037223815917969, -0.44610482454299927, 1.6586062908172607, 0.8453873991966248, 0.07845791429281235, -0.3365510106086731, -1.366443157196045, -1.1260898113250732, 0.5018501281738281, 1.7048828601837158, 1.0993757247924805, -0.28063255548477173, -0.6158408522605896, 0.013462993316352367, -1.2268836498260498, 0.3797207474708557, -0.27979570627212524, -0.8929638862609863, -1.0937522649765015, 0.00902189128100872, 0.05385682359337807, 1.9459925889968872, -1.0775700807571411, 0.07431832700967789, 0.2401009351015091, -0.16948722302913666, 0.21716570854187012, -0.21125593781471252, -0.5895723104476929, -0.05130773410201073, 3.0253376960754395, 1.5898948907852173, -0.8812888264656067, 1.0976389646530151, -1.8349189758300781, 1.6738359928131104, -0.9369019865989685, 1.3864901065826416, 0.35563531517982483, -1.157346487045288, 0.7847741842269897, -1.4810434579849243, 0.2518503665924072, 1.6389023065567017, -1.7606201171875, -0.011778981424868107, 0.37332460284233093, 1.031119704246521, 2.356478452682495, -0.6499766707420349, 0.6325485110282898, -1.5663447380065918, -0.9324867129325867, -0.04329516738653183, -0.19216878712177277, 1.179086685180664, 0.6518532633781433, -0.5826380252838135, 0.2351338267326355, -0.8775812983512878, -0.2804148197174072, -0.3229631781578064, 0.4629261791706085, -2.6730151176452637, 1.9691221714019775, 0.41716268658638, -1.7705763578414917, -0.5322444438934326, -0.015289943665266037, 0.8178749084472656, -0.47222062945365906, -1.1204472780227661, -1.2792699337005615, -0.35308873653411865, -1.180633783340454, 1.2048451900482178, -0.17788216471672058, 0.21171772480010986, -0.9212720394134521, 2.2125489711761475, -0.5796451568603516, -0.7387146353721619, 1.7659692764282227, -0.59102863073349, 0.5423975586891174, 0.5436680316925049, 0.32422661781311035, -1.740206241607666, 0.4660783112049103, -0.5128495097160339, 2.2719695568084717, -1.8933988809585571, 0.1503610461950302, -1.930046796798706, 1.3880749940872192, -0.7656837105751038, 1.0544536113739014, 0.5616397261619568, 0.6978131532669067, -0.2341252565383911, 0.029622556641697884, 1.135204792022705, -0.018624017015099525, -0.939243495464325, 1.8200465440750122, -0.4965376555919647, -0.48485273122787476, 1.0887508392333984, 1.023885726928711, 1.0522127151489258, 0.6751818656921387, 0.5929131507873535, 1.1826395988464355, 1.8884093761444092, -0.3583749532699585, -0.16269610822200775, -1.350034236907959, -0.6432151198387146, 0.568082332611084, 0.22203464806079865, -0.16938932240009308, 0.35463374853134155, -2.112325668334961, -0.15432128310203552, -1.0337122678756714, -0.41791462898254395, -0.15440598130226135, -2.0038421154022217, -0.9964045882225037, -0.4283839762210846, 0.00394462700933218, 0.5212889313697815, 0.808070957660675, 0.009039920754730701, 0.6892815828323364, 0.3527131676673889, 0.9240713715553284, -0.4876890480518341, 0.4371691942214966, -0.3782041668891907, -0.8958970904350281, -0.09310904145240784, -1.4936188459396362, -0.25912636518478394, -0.6541671752929688, -0.3436393141746521, -0.40100720524787903, -1.0907179117202759, 0.5582043528556824, 0.09150227904319763, -2.0334255695343018, -0.8959141969680786, -0.45301932096481323, 1.2239668369293213, -1.3472025394439697, -0.5206704139709473, -0.5403875708580017, 0.2048500031232834, 0.27994054555892944, -0.9317725896835327, -0.7594104409217834, -0.5492115020751953, -0.6883296966552734, 0.8308882713317871, -1.1596965789794922, -1.3428738117218018, 1.1494086980819702, 1.3519309759140015, -0.45938840508461, 0.49495387077331543, -0.8963717818260193, 1.958585262298584, -1.1620866060256958, -0.7864134311676025, -0.7055685520172119, -0.6751287579536438, 1.320719599723816, 0.5692773461341858, 0.834230899810791, -0.7214467525482178, 1.5326952934265137, -1.0437402725219727, 1.4725689888000488, -0.9955326318740845, -0.7622725963592529, -0.5125476121902466, 1.6662033796310425, -1.0876736640930176, 2.5733492374420166, 0.20148035883903503, 1.0776787996292114, -1.285622000694275, -2.530827760696411, -0.5043826699256897, 0.7323295474052429, -1.1805634498596191, -0.09245085716247559, 0.3477412164211273, 0.4480994641780853, -1.1020126342773438, 0.6715113520622253, 0.21736489236354828, -0.41158515214920044, 0.07457458227872849, -1.391711950302124, -0.2843720614910126, -0.21854615211486816, 0.4162614047527313, -0.752957820892334, -1.4254249334335327, -0.037191241979599, 2.8202409744262695, -0.8626483678817749, -0.25702929496765137, -0.249464750289917, 0.2738524079322815, -0.6638110280036926, 0.6306988596916199, 0.3941073715686798, 0.34137994050979614, 0.7544434666633606, 1.4007580280303955, 0.2751062214374542, -0.8752241730690002, 1.4193425178527832, -1.1661828756332397, 0.07070138305425644, -0.6690141558647156, -0.08660300076007843, 0.5609278082847595, 2.6945178508758545, 1.094774842262268, -1.8779990673065186, -0.048016276210546494, 1.082927942276001, 0.4404067099094391, -1.2676308155059814, 0.2851555645465851, 1.1313648223876953, -0.8946636319160461, -1.18646240234375, 0.3599866032600403, 0.363607794046402, 0.9259792566299438, 0.31909725069999695, -1.4935511350631714, 1.690995454788208, -1.251725435256958, -0.5850920081138611, 1.3979469537734985, -0.21823103725910187, -1.3020299673080444, 0.8107182383537292, -0.4630073606967926, 1.02885103225708, -0.6495471596717834, 1.3895249366760254, -2.735128164291382, 0.012562333606183529, -2.328585386276245, 0.5921863317489624, 0.7926493883132935, 0.9431992173194885, 0.8544673919677734, 0.991835355758667, 0.3058427572250366, 0.023155774921178818, 0.8018847703933716, -0.8684500455856323, 0.8262579441070557, -0.5484486818313599, -0.030628304928541183, 0.584212064743042, -1.7609531879425049, 0.500651478767395, 1.2760038375854492, 0.09447477757930756, 0.9780438542366028, 0.30810049176216125, -1.152012586593628, -1.3754247426986694, 0.05657454952597618, -1.2422003746032715, -0.008606234565377235, 1.0085110664367676, -0.2774524390697479, 0.8329749703407288, -0.43420514464378357, -0.634247899055481, 0.3975342810153961, 2.0369443893432617, -1.3616141080856323, 0.589180588722229, 0.650978684425354, -0.10667769610881805, 0.5745212435722351, -0.7809728980064392, 0.5677770376205444, 2.2874197959899902, 1.2485102415084839, 1.599632978439331, -0.13054899871349335, -0.22084401547908783, -1.7758382558822632, -1.766487717628479, 0.8180784583091736, -0.6676297187805176, 0.4273722767829895, 1.970367670059204, 1.7024208307266235, -0.40768927335739136, -1.3289554119110107, -0.3356482684612274, 0.16305413842201233, 0.8012586832046509, -0.13475416600704193, 2.2676892280578613, -0.7098914980888367, 1.9873682260513306, -3.4404900074005127, 1.0146121978759766, 1.4376906156539917, 0.42058953642845154, -0.26688963174819946, 0.1951419562101364, 0.1630958467721939, 1.1028571128845215, 1.3316309452056885, 0.5705180168151855, 2.9906487464904785, -0.8781279921531677, -0.33128082752227783, -0.5268105268478394, -0.6947612762451172, 0.12339720875024796, 1.1782455444335938, -0.3463731110095978, 0.24551144242286682, 0.11987432092428207, 0.28186920285224915, 0.4162399172782898, -1.148659110069275, 0.2242060899734497, 0.01594296470284462, -1.2297511100769043, 2.075488567352295, 0.5023795366287231, 0.9768792986869812, -0.9018629789352417, -2.3877477645874023, -1.3773995637893677, 1.532044529914856, 0.7345921397209167, -0.2945846915245056, -0.30321767926216125, -0.04459621012210846, 0.5516239404678345, -0.005974777042865753, -0.14160269498825073, 0.5588753819465637, 0.2057444006204605, 0.4947228729724884, -0.8631414771080017, 0.3754322826862335, 0.28130877017974854, -1.3963847160339355, 0.8943023085594177, 1.6502269506454468, -1.4288158416748047, 0.557659924030304, 0.9249348640441895, 1.029801845550537, 0.3378888964653015, -0.474334716796875, -0.6484590172767639, -0.07548154145479202, 0.9142074584960938, -0.031631577759981155, 2.468751907348633, -0.5957289338111877, 1.5589286088943481, 0.7451454997062683, -0.6755172610282898, -0.3628286123275757, 0.40297016501426697, 0.17655470967292786, 0.8776566982269287, -0.6593489050865173, -0.2539486289024353, 0.9633957743644714, 0.8488543033599854, 1.6126141548156738, 1.2391438484191895, 0.26733922958374023, -1.4673755168914795, 1.2773631811141968, 0.06375836580991745, -0.5437570214271545, -0.34832340478897095, 1.1135783195495605, 0.008744129911065102, -1.4457805156707764, 0.08851253241300583, 0.4744032621383667, 0.08341573178768158, -0.8717641830444336, -2.158907651901245, -0.995433509349823, 0.8518895506858826, -0.5515512824058533, 0.08742381632328033, -0.3057456612586975, 0.7586574554443359, -0.5877099633216858, -0.6334714889526367, -1.2470016479492188, -0.2768179476261139, -0.050770822912454605, -0.8260273337364197, 1.0483038425445557, 1.1556779146194458, -0.7107570171356201, 0.614109218120575, 0.5994019508361816, -0.01749640889465809, 0.27671387791633606, 3.2519774436950684, 0.10085820406675339, 0.18247908353805542, -1.9985551834106445, 1.2417323589324951, 0.15267573297023773, 0.789804995059967, -1.295873761177063, 0.8160449862480164, 0.6800913214683533, -0.27914807200431824, 1.1761622428894043, 1.5982513427734375, -0.18751104176044464, 0.03714048117399216, -2.43967604637146, 0.6087958812713623, 0.4756593108177185, -1.0040727853775024, -0.8992509245872498, 0.3522355556488037, 0.4712386131286621, 0.4811570644378662, -0.024751536548137665, 1.746504545211792, -0.08742565661668777, 0.5608022809028625, -0.8482595086097717, 0.33513495326042175, 0.25596630573272705, -1.0344659090042114, 1.0050718784332275, 0.4830555021762848, -0.06458637118339539, -1.3250638246536255, -0.5268017053604126, -0.4250562787055969, 1.7933675050735474, -0.18020601570606232, -1.7059473991394043, -0.838746190071106, 0.3088683485984802, 0.3207932412624359, 2.8953099250793457, 1.5136553049087524, 1.057511329650879, -0.4198564887046814, 0.3379521369934082, 0.43304920196533203, -1.2039122581481934, -2.1388401985168457, 0.8317506909370422, -0.8773427605628967, 0.4901539385318756, -0.3471662104129791, -0.19397497177124023, 2.541181802749634, 0.49999895691871643, 1.7867119312286377, -0.7743534445762634, -0.8420729041099548, 0.11270270496606827, -0.5233561396598816, 0.05943930521607399, -1.0714020729064941, 1.0656681060791016, -1.2766287326812744, -0.2409522980451584, 0.22961540520191193, 1.9132088422775269, -0.41258788108825684, -1.888929009437561, -0.09213168919086456, 1.365740418434143, -0.16461515426635742, 0.23856595158576965, -0.06774075329303741, 1.3161331415176392, 0.4811256527900696, -2.1473851203918457, -0.12677167356014252, -0.9390947818756104, 0.5217241644859314, 0.6109693050384521, -0.19053173065185547, 0.03183495253324509, -0.001353073283098638, -2.7080399990081787, 0.25739771127700806, 0.6947602033615112, 0.4313284456729889, 0.21254850924015045, 0.1117398738861084, -0.9296103715896606, -0.7745314836502075, -0.40211614966392517, -0.5009007453918457, 0.38415032625198364, 1.9141536951065063, 0.3691282272338867, -0.24091999232769012, 0.17888037860393524, 1.7224751710891724, 0.20034272968769073, 0.3463899493217468, -0.2522451877593994, -0.5852857232093811, 0.5591357946395874, -0.5848774909973145, -0.05708054080605507, 0.7891103625297546, 0.14890654385089874, -1.3981446027755737, -3.236659526824951, -0.42267394065856934, -0.37108075618743896, 0.2983447313308716, 1.2633837461471558, 0.9456549286842346, 0.25461968779563904, -0.4730939269065857, 0.12986846268177032, 0.6471060514450073, 0.34185758233070374, -0.13383525609970093, -0.96208256483078, 0.048558201640844345, -1.1270153522491455, -2.884237766265869, 1.185043215751648, -0.2573561668395996, -0.09193883836269379, -2.006115436553955, -0.8879702091217041, -1.3590211868286133, -1.0662785768508911, -1.043818473815918, -3.2644169330596924, 0.49476903676986694, 0.35107916593551636, 0.5311805605888367, -0.43995437026023865, 1.0571213960647583, 0.5197288990020752, -1.475274920463562, -0.48972445726394653, 0.3719295859336853, -1.5381532907485962, -0.03716767951846123, -0.5694352388381958, 0.8752593994140625, 0.6480090022087097, -0.34251800179481506, -0.2903658151626587, -0.0690147653222084, 0.9323695302009583, -0.8152280449867249, -1.8162461519241333, -0.031942255795001984, 0.06810404360294342, -1.369468331336975, -1.797878623008728, -0.5289672017097473, -0.18908242881298065, 0.3955745995044708, -2.9780209064483643, 0.3831847906112671, -0.9839147925376892, 0.2650323808193207, -0.9039272665977478, -0.6659967303276062, 0.026220185682177544, 0.06704013049602509, 0.8722589612007141, 2.934631109237671, 1.5679632425308228, -1.283568263053894, 0.027210446074604988, 0.8706064820289612, 0.2656424343585968, 0.4720029830932617, 1.3929015398025513, 0.7509042620658875, 0.5379635095596313, -0.23342151939868927, -0.8401814103126526, -0.49297434091567993, 0.6255249977111816, 0.376608669757843, 0.30474603176116943, 1.9760890007019043, -0.17666028439998627, 1.3238543272018433, -0.6138001084327698, -0.21814092993736267, 1.3916840553283691, 0.7496208548545837, -0.35024821758270264, -1.144408941268921, 0.003622794058173895, 1.3502182960510254, -0.23885290324687958, 1.0159156322479248, -1.0598831176757812, 0.9168453216552734, 0.12352262437343597, -0.6178640127182007, 0.08655470609664917, -0.38996732234954834, -0.6555100083351135, 0.043210241943597794, 0.43401604890823364, 0.6827839612960815, -1.5765316486358643, 0.6917763948440552, 0.43263211846351624, -1.2536673545837402, -0.3320256769657135, 2.4947257041931152, -1.2192983627319336, 1.1000163555145264, -0.622911810874939, 2.6535685062408447, 0.6417933702468872, 0.5713503360748291, -0.6966535449028015, 1.4489091634750366, -0.02934054099023342, -2.600552797317505, -0.9081827998161316, -0.8284318447113037, 0.25392958521842957, 0.1779385507106781, 0.6697443723678589, 0.44512638449668884, -0.2824952304363251, -0.46552419662475586, 1.8896288871765137, 0.3035643994808197, -0.21900850534439087, -0.3708359897136688, 0.7202669978141785, -0.29371365904808044, -0.34343433380126953, -0.7656282186508179, -1.2387248277664185, 0.13692685961723328, 2.1603775024414062, 0.3943789601325989, 0.7072067856788635, -0.2656556963920593, 2.807731866836548, -1.8700472116470337, -1.247055172920227, -0.8962850570678711, -0.31375813484191895, -0.40401697158813477, 0.5038228034973145, 0.06456095725297928, -1.234355092048645, -0.6691901683807373, -0.3458818793296814, -0.6018064022064209, -0.24173352122306824, -0.7584022879600525, -0.6147647500038147, -0.4375819265842438, 0.6562528610229492, -2.1331706047058105, 0.8059780597686768, 1.0950931310653687, 0.9694651961326599, -0.9256632924079895, 0.6334653496742249, -0.6886805891990662, 1.2993857860565186, -0.18209344148635864, 0.6486589908599854, -1.1435223817825317, -0.7763862609863281, 0.2771739959716797, 0.2793910503387451, -1.004124641418457, -0.5664456486701965, 0.7652749419212341, -0.019452938809990883, 0.9640042781829834, -1.3444989919662476, -0.8233258724212646, -0.7493588328361511, -1.3951181173324585, 0.24837476015090942, -0.6368250846862793, -0.36716848611831665, 2.038935661315918, 0.053268913179636, 0.772155225276947, 1.2985343933105469, 0.33517998456954956, 0.30416110157966614, -0.7585071921348572, -0.6580480933189392, 0.058177608996629715, 1.2456868886947632, 0.9714586138725281, -0.6253857016563416, 0.272164523601532, -1.865344762802124, 2.4780287742614746, -0.06183728203177452, 1.079399824142456, -0.15234051644802094, -0.2387138456106186, 0.743054211139679, 0.03755921125411987, 0.5832957625389099, 0.9424247741699219, 0.6175352334976196, 0.3666273355484009, -0.09035459160804749, 0.44546008110046387, -1.469763994216919, -0.19523657858371735, 1.380185842514038, 1.1436021327972412, 0.33020102977752686, -0.4581722915172577, 0.3383292555809021, 0.4699456989765167, -0.29168280959129333, -1.1399500370025635, -1.6821972131729126, -0.35525548458099365, -0.4903242290019989, -1.0670429468154907, 2.1625726222991943, 0.3971828520298004, -0.1825246512889862, 0.6826568841934204, 0.1927863508462906, -0.3565729856491089, -0.2976512014865875, -0.3345758020877838, 0.7097728848457336, 0.027800031006336212, 1.9442416429519653, -0.701564371585846, 1.1456127166748047, 0.9127652645111084, 1.4400595426559448, -0.08546143770217896, -0.8731671571731567, -0.2604043781757355, 0.26938241720199585, 2.7910618782043457, -0.96267169713974, -0.8332553505897522, 0.3680519163608551, 2.070528268814087, -0.10126709938049316, -1.7710739374160767, 1.0796705484390259, -0.586489200592041, -0.2818605601787567, -1.1254693269729614, -1.287623405456543, 0.7325845956802368, 2.178241729736328, 0.6494305729866028, 0.39645105600357056, -0.3446178734302521, 0.28017812967300415, -0.04497556388378143, -0.7963460683822632, -0.23357675969600677, 0.5812296867370605, 1.124502182006836, -0.5488139986991882, 0.09966090321540833, 1.7284724712371826, -0.25146251916885376, -0.9694119691848755, -0.8705052733421326, 0.5669983625411987, 1.8226523399353027, 0.09214954078197479, 0.6943396925926208, -0.897779107093811, -2.218522071838379, -0.7360557317733765, -0.7423680424690247, 0.1705654263496399, 0.3764338195323944, 1.3257756233215332, 0.34322720766067505, -1.0446363687515259, -0.15518228709697723, -0.15248319506645203, -0.6660963296890259, 2.2135729789733887, 1.144312858581543, -1.5314832925796509, 0.508898913860321, 0.9613644480705261, 1.715200662612915, -0.7538236379623413, -0.039431508630514145, -0.1161169484257698, -0.018498620018363, 0.9621596932411194, 0.2913144826889038, 0.25011301040649414, -1.2946739196777344, 1.4210586547851562, 0.8146181106567383, 0.4491499364376068, -1.5608068704605103, -0.24853456020355225, 2.410890579223633, 0.09462888538837433, 0.20791782438755035, -0.4805871844291687, 1.0777186155319214, 1.638831615447998, 1.2344001531600952, -0.2150331288576126, -0.6396616697311401, 0.28400447964668274, -1.118635654449463, -0.8639245629310608, -0.43285226821899414, 1.2146081924438477, 0.002004266018047929, 0.8701013326644897, -1.8387722969055176, 1.3291879892349243, -1.3818295001983643, 0.3656473159790039, -1.283686637878418, 0.7553377747535706, -1.0623832941055298, -1.5322080850601196, 0.13256923854351044, -0.30838990211486816, -0.09998225420713425, 1.2744978666305542, -1.6975765228271484, 0.9044251441955566, 0.43597760796546936, -0.08205267786979675, 0.7477833032608032, 1.242571473121643, -0.2932656705379486, -0.245614156126976, -0.8877925872802734, 0.8301988244056702, -1.9734441041946411, 1.855094313621521, -0.15062318742275238, -0.09509502351284027, 0.873960554599762, 0.8549960851669312, 0.8459774255752563, 0.05916893109679222, 2.0573253631591797, -0.748338520526886, 0.49546298384666443, 0.17384491860866547, 0.4748014211654663, 0.867651104927063, -0.31527218222618103, 0.11415354907512665, 0.15126238763332367, -0.4690072238445282, -0.6508928537368774, -0.3674915134906769, -0.892035722732544, 0.0073199085891246796, -1.512371301651001, -1.583984136581421, -0.5457629561424255, -0.5305339097976685, 0.048204440623521805, -2.0619142055511475, -0.9815773963928223, 0.8372617363929749, -0.23661407828330994, 1.0709885358810425, -0.1625741571187973, -1.126128911972046, -0.5250599384307861, 0.2363043874502182, 1.4824047088623047, 0.42895185947418213, 2.324723482131958, -0.1477147489786148, 2.7559397220611572, 1.592687726020813, -0.36722174286842346, 1.201673984527588, -2.151909351348877, 0.01551717147231102, -1.2779380083084106, -0.8656406998634338, -0.7068541646003723, 0.5440430641174316, 0.5693144202232361, 1.0410847663879395, 1.1744030714035034, 0.7083932757377625, -0.20851892232894897, 0.016647327691316605, 0.6526951193809509, -0.42985114455223083, 0.41357529163360596, -0.22194236516952515, 0.5273691415786743, -1.5117101669311523, -0.12849386036396027, -0.24779994785785675, -1.2792282104492188, 0.6251918077468872, 1.2904438972473145, 1.5318140983581543, 2.1858484745025635, 0.5141278505325317, -0.9540798664093018, 1.1603351831436157, 0.1466986984014511, 1.4219095706939697, 0.879364013671875, 1.5714876651763916, -0.0641849935054779, 0.12427960336208344, -1.3517482280731201, 0.6384266018867493, -1.0776394605636597, -0.18135683238506317, -2.278780698776245, 0.39391928911209106, 0.8694117665290833, 1.6541008949279785, 0.05674755573272705, 0.8586269021034241, -0.4814856946468353, 0.3955405652523041, 0.7248467206954956, -1.2850269079208374, -0.43045341968536377, 0.7749866843223572, -0.6682711839675903, -1.0397915840148926, 0.9430465698242188, -1.7604247331619263, -1.1822209358215332, -0.188938170671463, -0.2945486605167389, -0.46884357929229736, -0.9145142436027527, 0.04948122054338455, -1.2255096435546875, -0.49663370847702026, 2.3714325428009033, -1.6205288171768188, 0.9857668876647949, 0.12330368161201477, -0.5031965970993042, 0.5311620235443115, 0.5415318012237549, 0.0831807479262352, 0.4681818187236786, 0.18254558742046356, -0.15777142345905304, -0.29445284605026245, 0.39655497670173645, 1.6613634824752808, -0.6346033811569214, -1.109932541847229, 0.8820720911026001, 0.4953044354915619, 0.8555769920349121, -0.103215791285038, 2.088428497314453, -0.03901268541812897, 0.0451289564371109, -1.2572143077850342, 0.43762972950935364, 0.8720124959945679, 1.382407784461975, -1.2201873064041138, 0.7517261505126953, -1.1578271389007568, -0.3306907117366791, -0.8824644088745117, -0.23698881268501282, -1.5898494720458984, -0.7062409520149231, -1.5127861499786377, -1.4686059951782227, -0.9251951575279236, 0.3163812458515167, 0.36534374952316284, 0.5576224327087402, -0.4233911335468292, 0.7490829229354858, -1.104095458984375, 0.3956204354763031, 0.6044902801513672, 0.9130794405937195, 0.6836671233177185, 1.2886539697647095, 1.2238608598709106, 0.6506466865539551, -0.15203352272510529, -0.8014387488365173, 1.0983376502990723, -0.20172321796417236, -0.41258320212364197, 0.4805472791194916, 1.1102012395858765, -1.1238631010055542, 0.4638976454734802, 1.5352952480316162, -0.256836473941803, 1.0569556951522827, 0.6696733832359314, -2.1199889183044434, 0.2840375006198883, 0.5720909237861633, -0.4899042844772339, -0.5140661001205444, 1.8023054599761963, 1.2873860597610474, 0.2525503933429718, -0.7743906378746033, 0.35759437084198, 0.06677164137363434, -0.9196540117263794, 0.18194551765918732, -0.31161099672317505, -0.07174603641033173, -0.5238351821899414, 0.9156054854393005, 1.4400684833526611, -0.09765686094760895, -0.21479427814483643, -1.0337536334991455, -0.10887453705072403, -1.2933675050735474, 0.1067335307598114, -0.48096951842308044, 0.1292465478181839, -0.45834994316101074, 0.3981558680534363, -0.03435659408569336, -1.366382122039795, 1.8083560466766357, -0.5378581881523132, -1.2227660417556763, 1.3722444772720337, -0.6342455744743347, -0.5626053810119629, 0.1413474828004837, 1.3773703575134277, -0.38029733300209045, 1.1732760667800903, -0.06962895393371582, -0.5120643973350525, 0.07421796768903732, -0.43447813391685486, 1.3332430124282837, -1.4684206247329712, -0.9690811634063721, -0.730807363986969, -0.8231068849563599, -0.276175320148468, -0.7809324264526367, 0.21716293692588806, 0.9849748015403748, 0.383571594953537, 1.2054483890533447, -0.876710832118988, -1.4250342845916748, 0.6102606654167175, 1.8153882026672363, 0.06622438877820969, -0.2520865499973297, 1.7864384651184082, 1.3872973918914795, 1.3062382936477661, -0.8840542435646057, 0.11274838447570801, 0.8877884149551392, 0.13668832182884216, 0.9421136379241943, -0.19994056224822998, -0.01238964218646288, -1.4122499227523804, 0.7242147326469421, -0.39806845784187317, -0.5221479535102844, 0.5367841124534607, -1.0172877311706543, 0.8057448863983154, 0.9710453152656555, -0.6424762606620789, -0.7416545152664185, 0.030571164563298225, -0.6153233051300049, -0.5370203852653503, -1.7546523809432983, -0.45019081234931946, -1.2732652425765991, 0.21295993030071259, -0.05957922711968422, 1.5344599485397339, 0.7083619832992554, -0.6663709282875061, -0.041572801768779755, 0.361100435256958, 0.4343782067298889, -0.34302178025245667, -0.19385093450546265, 0.9627683162689209, -1.1223599910736084, -0.34941038489341736, 1.5625373125076294, 1.0994144678115845, -2.0720112323760986, -0.5960181951522827, -1.1086158752441406, 0.037649061530828476, 0.15178550779819489, 0.2983713746070862, -0.12182961404323578, 0.7350504994392395, 1.5526398420333862, -1.593743920326233, -0.900098979473114, -1.5572770833969116, -1.0226333141326904, 0.2574984133243561, 1.4888672828674316, -1.4147042036056519, 0.9182530045509338, 1.7677441835403442, 0.3380424380302429, -0.6507638692855835, 0.9752421975135803, 0.19292762875556946, 0.4500740170478821, 1.7376751899719238, -1.0377025604248047, 1.2946186065673828, 0.7306241989135742, 0.8460853695869446, -0.578927218914032, -0.13539686799049377, 0.3437611758708954, -0.40323886275291443, -0.6163045763969421, -0.09271115809679031, -2.691039800643921, -1.1761770248413086, -0.7010757327079773, -0.6478078961372375, 1.04152512550354, 0.24581648409366608, -0.5617290735244751, -1.474036693572998, -1.446663737297058, -1.27853524684906, -0.6641950607299805, 0.023447003215551376, 1.2516478300094604, 1.092345118522644, -0.39148399233818054, 0.3649143576622009, -0.33362460136413574, 0.139659121632576, 0.481925368309021, 0.6439937353134155, 0.4235137104988098, 1.5694375038146973, 1.1704941987991333, -1.7920366525650024, -0.41295865178108215, -0.7192502617835999, 0.7096297740936279, -0.874178946018219, -1.0291967391967773, 0.5100562572479248, -2.5541396141052246, -0.6079098582267761, 0.7151321768760681, 0.4387178421020508, -1.7989100217819214, 0.5855785012245178, -1.180893063545227, -0.5006892085075378, -0.8028594851493835, 1.5835832357406616, 0.16638080775737762, -0.40059977769851685, 0.20442146062850952, 0.13789871335029602, -0.27278220653533936, 0.9981268048286438, -0.898318886756897, 0.2892517149448395, -0.19801077246665955, 0.18654561042785645, 0.8457605242729187, 1.3720906972885132, -0.6046403646469116, -0.4167726933956146, -0.9465067386627197, -1.9327964782714844, 2.200571298599243, 0.7371566891670227, -2.083524465560913, -0.9903407096862793, 0.657471239566803, 1.2136956453323364, -0.731645405292511, 0.7213078737258911, -1.6726301908493042, -0.10778779536485672, -0.4159485399723053, -0.9478669762611389, -2.010185718536377, -0.7060818076133728, -1.1526358127593994, -0.7990084886550903, 2.9690792560577393, 0.1754421889781952, -0.41509583592414856, 1.4999682903289795, 1.0343921184539795, 0.28785309195518494, 0.4504587948322296, 1.958514928817749, -1.0732219219207764, -0.9384247660636902, -0.5449578166007996, 1.3759229183197021, -0.8702929615974426, -0.6893106698989868, -1.133590579032898, -2.3677592277526855, -1.42583167552948, -0.4636048376560211, 0.13790318369865417, -1.5069758892059326, 1.0644358396530151, 1.1610840559005737, 0.6374610662460327, 1.0913251638412476, -0.2524682581424713, -0.25253432989120483, -0.20655441284179688, 1.0374506711959839, 0.332121878862381, 1.2607712745666504, 0.8573312759399414, 1.9123435020446777, -1.0890247821807861, 0.27809059619903564, 0.370131254196167, -0.2551259398460388, 1.7338550090789795, 0.0015613202704116702, 0.807296097278595, 0.7553043961524963, 0.36728382110595703, -1.6800004243850708, 0.6289582848548889, -0.4008908271789551, 0.8738171458244324, -1.0463374853134155, 1.2689845561981201, 0.26576343178749084, -0.9227412343025208, -1.605378270149231, 1.644256591796875, -1.78407621383667, 0.20754554867744446, 0.6447615027427673, 0.24288149178028107, -0.6905638575553894, 0.846184492111206, 0.8259817361831665, -0.6783069968223572, 0.73187255859375, -1.4043607711791992, -0.05984244868159294, 1.399353265762329, 0.06902909278869629, -0.4226248860359192, -0.60931396484375, 1.4488630294799805, -0.08418840169906616, 0.27990251779556274, -0.8847619891166687, -0.7505688071250916, 0.30351853370666504, -2.4029369354248047, -0.33478739857673645, 0.6820784211158752, -0.9088311195373535, 1.4960969686508179, 0.8730770945549011, 0.16617295145988464, 0.7253274917602539, -0.888817548751831, 1.6178326606750488, 1.7134705781936646, -1.2638744115829468, -0.2889918386936188, 1.2102779150009155, -0.0148318475112319, -0.2998064458370209, 0.5736985802650452, -0.9328187108039856, -0.35237064957618713, 0.7661028504371643, -1.2489842176437378, 0.859058141708374, -0.1959162950515747, 1.4480067491531372, -1.2588412761688232, -1.915115237236023, -0.12558777630329132, 0.7424808144569397, 0.22604061663150787, 0.3562653064727783, 0.6656924486160278, 1.7904187440872192, 0.7133404612541199, 0.5747907161712646, -1.133607268333435, -1.8824692964553833, -1.0760127305984497, -0.4977339506149292, -0.8378555178642273, 1.04770827293396, 0.2673450708389282, 0.8276536464691162, -1.3827263116836548, 0.8057337999343872, 0.2581728994846344, -0.4621685743331909, 0.45866385102272034, 0.2672533690929413, 1.0877351760864258, 0.37413808703422546, -1.4168809652328491, -0.28353261947631836, -0.17247913777828217, -0.08487199991941452, 0.35422590374946594, -1.1010420322418213, -1.021506667137146, -0.5537379384040833, -0.703863263130188, 0.8845812082290649, -0.20730580389499664, 0.8643648028373718, -0.25991737842559814, 0.7423173189163208, -0.04397124797105789, -0.8578015565872192, -1.0880465507507324, -0.04163702577352524, 1.6908735036849976, -0.1557493507862091, -0.42069560289382935, -1.6278650760650635, 0.6149244904518127, 0.34580910205841064, -1.694707989692688, 1.365354299545288, 0.5251456499099731, 0.9675611257553101, -0.17169976234436035, 0.12611399590969086, -0.3721948266029358, 1.5422903299331665, -0.32838428020477295, 0.467267245054245, 0.8082560300827026, -1.3537052869796753, 2.1906237602233887, -0.8953049182891846, 0.3595046103000641, 1.1019333600997925, 1.4775536060333252, -1.8250682353973389, -0.5238676071166992, 1.620073676109314, -0.47906917333602905, 0.07655983418226242, -0.06388608366250992, 1.3138478994369507, -0.9327305555343628, -0.7307640910148621, 0.4989672601222992, -0.0038535657804459333, -1.429104208946228, 0.8430569767951965, -0.5022851824760437, -0.36646175384521484, -0.9549401998519897, 0.6149780750274658, -0.3893008232116699, -0.43155157566070557, -0.6674374938011169, 1.3852227926254272, -1.0992710590362549, 1.3599889278411865, -0.17995581030845642, 0.17850598692893982, -0.8654782176017761, 0.002813511760905385, 0.01746441423892975, -0.011491650715470314, -1.351400375366211, 0.12678536772727966, -1.400456190109253, 1.338422179222107, 0.48845311999320984, 0.21701931953430176, 0.9042980670928955, -0.3399394750595093, -0.5193068981170654, 0.7634779810905457, -0.33324694633483887, 1.6458982229232788, 0.0258611049503088, 0.9355031251907349, 0.7687302827835083, -0.6128727793693542, -0.48494961857795715, 1.0705227851867676, -0.13455601036548615, -0.0933089330792427, -0.44128355383872986, 0.578326404094696, 1.2443293333053589, -1.3764065504074097, -0.4546332061290741, -0.7787817120552063, 0.6937719583511353, 1.019477128982544, 0.496611088514328, 0.340623676776886, -0.9260802268981934, -1.016682744026184, 0.5683107972145081, -0.8914915323257446, 0.14954839646816254, -0.7049671411514282, -0.574394166469574, 0.4102502763271332, 0.7781442999839783, -0.48141682147979736, 1.2408957481384277, -2.3968822956085205, -0.8400547504425049, -0.2696072459220886, 1.0777721405029297, 0.07079049944877625, 0.8306252956390381, -0.4995407462120056, 1.3372843265533447, -0.031495507806539536, 0.6225089430809021, -0.8917463421821594, -0.4310557246208191, -0.21589581668376923, 1.6596230268478394, -0.1059584990143776, -0.5019925236701965, -1.358066201210022, -0.009161793626844883, -0.44871318340301514, 0.6654516458511353, 0.19718872010707855, 0.9908804297447205, 1.3645200729370117, -1.66499662399292, 0.5359081029891968, 0.21835866570472717, 0.23855140805244446, 1.0100815296173096, -2.3798391819000244, 0.9898349046707153, -0.18042877316474915, 0.7702791094779968, 0.40418869256973267, 0.45738592743873596, 0.8650065660476685, -0.20028162002563477, -0.22879694402217865, 0.5237828493118286, -0.7779150009155273, 0.8036746382713318, -0.8889484405517578, 0.11793556809425354, 1.6885888576507568, 0.7270112633705139, -1.3138937950134277, 0.08228973299264908, 1.0060416460037231, 0.2956957221031189, -1.0417159795761108, -0.5863679051399231, -1.0602463483810425, -0.43986019492149353, 0.6786286234855652, -0.3397332727909088, -0.7025966048240662, -2.0060389041900635, -0.1192978098988533, 0.18821895122528076, -0.1428719013929367, -0.4869031608104706, 0.6481332778930664, 0.7375412583351135, 0.3383292853832245, 0.4875209331512451, -0.6397995352745056, 1.0277608633041382, 0.9111039638519287, -0.39328914880752563, -0.5259580016136169, -0.5587117671966553, 0.811357855796814, -0.26557233929634094, -0.44361409544944763, 1.5638012886047363, -0.3220078647136688, 0.6909499764442444, -0.5361567735671997, 1.3559409379959106, 0.22787104547023773, -2.24231219291687, 0.18290357291698456, -0.31875330209732056, -1.1120327711105347, 0.5297223329544067, 1.1721397638320923, -0.27124151587486267, -1.5179312229156494, 2.6055073738098145, 0.5299402475357056, 0.8194693326950073, -0.07617561519145966, 3.2913880348205566, 0.8003538846969604, -0.43334823846817017, 0.44338756799697876, -0.6369744539260864, -1.5292555093765259, -0.3376024663448334, 0.7411221265792847, -0.196807399392128, 0.08500656485557556, 3.0693905353546143, 0.13100431859493256, 0.7707178592681885, -0.039449293166399, 0.1099071055650711, 0.7701690196990967, 0.9630838632583618, -1.0147833824157715, 0.42043086886405945, -0.7878987789154053, 0.826465904712677, -0.00996309146285057, 1.3274656534194946, 1.4577046632766724, 0.0010958312777802348, -0.33635419607162476, 0.5473840236663818, 1.4263566732406616, 2.046614408493042, 0.9779314398765564, -0.061080340296030045, 0.52345871925354, 1.4174327850341797, -1.8621711730957031, -1.047236680984497, 2.5030198097229004, 0.5480148196220398, -1.371826171875, -0.21831178665161133, 1.472079873085022, 0.3329843580722809, -2.0459957122802734, 0.9633486866950989, 1.5954694747924805, 1.2378358840942383, 1.0942600965499878, 1.6264077425003052, 0.15516890585422516, 0.45063847303390503, 0.7063334584236145, 0.034517545253038406, 1.789541244506836, -0.07498877495527267, -0.6804020404815674, -0.11043678969144821, 0.916438102722168, 1.2424105405807495, 0.2576638162136078, 0.32101911306381226, 0.8028551340103149, -0.0888328105211258, 0.4987819194793701, -0.1904015690088272, 0.9700416922569275, -0.1884269416332245, 1.2852438688278198, -0.8068927526473999, -0.4482598900794983, 0.7359937429428101, 0.1856009066104889, -0.1321808248758316, 1.3689332008361816, 2.7043356895446777, 0.4151212275028229, -0.683295488357544, 0.029668301343917847, 1.321748971939087, 0.15866678953170776, -0.32037585973739624, -0.9161102175712585, 1.6023046970367432, -0.14040511846542358, 0.11135151982307434, 0.4193763732910156, 0.6520407199859619, 0.7525503635406494, 1.7558106184005737, 0.04779139533638954, -0.07321300357580185, 1.2497992515563965, -1.122794270515442, -0.9979522228240967, 0.3781026303768158, -0.3735138177871704, 0.718788743019104, 1.185333013534546, -0.41532355546951294, -0.13423727452754974, -0.6598539352416992, 0.8471766114234924, -1.4497699737548828, -0.9016344547271729, 0.8544948101043701, -0.8206877112388611, 1.0139967203140259, 0.7001307010650635, 0.6626254320144653, -0.3941114544868469, -0.4773959815502167, 0.41730889678001404, 0.29045236110687256, 0.5648941993713379, -0.160626620054245, -0.050696562975645065, 0.47446075081825256, -0.9079801440238953, 1.314400315284729, 0.746342658996582, 2.117709159851074, 0.5396052598953247, 0.20986731350421906, -0.7613154053688049, -0.52772456407547, 0.3818973898887634, -0.6693539023399353, -1.7530279159545898, 0.5620514750480652, 1.2139309644699097, 1.2134861946105957, -0.5670505166053772, -0.5816474556922913, 0.20574840903282166, -1.163684368133545, -0.3968208432197571, 0.06413709372282028, 0.254107803106308, 0.5528077483177185, -1.7558457851409912, 1.4073964357376099, 1.3187581300735474, -0.21667686104774475, 0.34801313281059265, -0.9430084228515625, 0.5351950526237488, 0.0727267935872078, 1.8183417320251465, 1.3199529647827148, 1.1985291242599487, 0.27247267961502075, 0.5655320882797241, 2.2435362339019775, 0.3326556384563446, -1.560786247253418, 0.4464927315711975, 0.4435127377510071, -0.5572549104690552, -0.4693205654621124, -0.30172061920166016, -0.6342882513999939, 0.6395822167396545, -1.124807596206665, -0.8531618118286133, 1.2071336507797241, 1.081618070602417, 0.8193563222885132, -0.1784590631723404, -1.1799718141555786, 0.12630905210971832, 0.12711496651172638, -0.8508651256561279, 1.6206409931182861, 0.7970332503318787, -0.018117131665349007, -0.41208383440971375, -0.8003345727920532, 3.0739517211914062, -0.2773374319076538, 1.3431813716888428, -0.42467623949050903, -0.8726112842559814, 0.3495660424232483, 1.8333816528320312, 0.22100992500782013, 0.9530061483383179, -0.9911995530128479, 0.3194952607154846, 0.2651768922805786, 1.2393858432769775, -1.1902666091918945, 0.5085002779960632, 1.1220664978027344, -0.4418807029724121, 0.2192913293838501, -1.0006206035614014, 0.838021457195282, 0.7025697231292725, 0.42063483595848083, -0.7379311323165894, -0.7088823318481445, 0.768134355545044, 1.2165824174880981, 0.09439793229103088, -1.104880928993225, 1.4170726537704468, -0.15848419070243835, 2.008776903152466, 0.3333250880241394, 0.14365145564079285, 1.7167409658432007, -0.8246092796325684, -0.49934449791908264, -0.6858195066452026, 1.9026470184326172, 0.9059478044509888, -1.5421127080917358, -0.48235252499580383, 0.4765941798686981, 0.6054137945175171, -0.45239830017089844, 0.8545050621032715, -1.9963257312774658, 0.9185309410095215, -0.34090456366539, 0.01824544370174408, -1.0198390483856201, 0.7949976921081543, -1.9637038707733154, 0.6716402173042297, -1.7858259677886963, -1.5068247318267822, -0.7602339386940002, -1.9065710306167603, -1.4554146528244019, -0.6439120769500732, -0.36046603322029114, -1.0275901556015015, 0.3090899884700775, -0.11356639862060547, 1.2525615692138672, -1.2061183452606201, 0.729995608329773, -0.7321957349777222, 0.8304445147514343, -0.293876588344574, 0.5637465119361877, -0.21946009993553162, 1.448895812034607, -0.3414573073387146, 0.523052990436554, -0.026957262307405472, -1.036847472190857, 0.9575844407081604, -1.2207306623458862, -0.7793827652931213, 0.7725741863250732, 1.335002064704895, -1.0764260292053223, 0.8846490979194641, 0.019067851826548576, 0.40550804138183594, 0.6026955842971802, -0.4033856987953186, -1.1816458702087402, 0.7286495566368103, 1.1371008157730103, -0.8298709988594055, -1.168777346611023, -1.9078736305236816, 1.0424998998641968, -0.6805914044380188, -0.3548152446746826, 0.444423645734787, -2.0331897735595703, 1.497496485710144, 2.2191390991210938, 1.087995171546936, -1.6188400983810425, 0.460734099149704, -0.4256461560726166, 0.9375272393226624, -1.5664474964141846, 0.14947184920310974, -0.6819789409637451, -0.383980393409729, -0.16149114072322845, -0.44110366702079773, -1.1484062671661377, -1.326528549194336, -0.23517051339149475, -0.6549341082572937, 0.2532193660736084, -0.66783607006073, -0.005111214704811573, -0.6052586436271667, 1.6229000091552734, 1.2401247024536133, 0.6759928464889526, 0.11588604748249054, 0.05713644251227379, -0.09946833550930023, 0.7532430291175842, 1.3423991203308105, 0.9793369174003601, -1.0761964321136475, 0.12972955405712128, 1.093435287475586, -0.3971034288406372, 0.13641121983528137, 1.3863978385925293, -0.097066231071949, -0.8994467854499817, -0.7596908211708069, -0.08469713479280472, 1.0474727153778076, -0.5459469556808472, -0.32528746128082275, 0.4959529638290405, 0.8673844933509827, -0.6567602753639221, -0.4493172764778137, 0.07829725742340088, -0.42420583963394165, -1.963645339012146, 0.6332525610923767, 3.187134027481079, 0.7914826273918152, 0.9980902075767517, 0.22002655267715454, -1.0425260066986084, 0.16979847848415375, -1.5616101026535034, -0.4775759279727936, -0.30301806330680847, 1.2443194389343262, 0.04205423966050148, -0.4361800253391266, -0.5237106084823608, 0.11342599987983704, 0.15255950391292572, 0.3418119251728058, 0.2016705870628357, -1.1191924810409546, 0.13764183223247528, 0.46184006333351135, 1.6509422063827515, 1.9100468158721924, -2.0479490756988525, -1.195798635482788, 1.1486413478851318, -0.5016000866889954, 2.1687207221984863, 0.43575459718704224, 1.0214189291000366, -0.4975423216819763, 1.5944305658340454, -0.48628950119018555, 0.06712799519300461, 1.256783127784729, -2.270684003829956, 0.5073159337043762, -0.21824447810649872, -0.46091705560684204, 0.7097218036651611, -0.36088982224464417, -0.37592071294784546, -1.4032822847366333, 1.1190420389175415, 2.546703577041626, 0.7107159495353699, -1.0429010391235352, -2.3637564182281494, -0.11252455413341522, 0.3907144069671631, -0.22160185873508453, -2.416947841644287, 1.2961657047271729, -0.20822662115097046, -1.0638391971588135, -0.1047806367278099, -0.2723923623561859, -1.4277474880218506, 0.5098581314086914, 1.4753161668777466, -0.34451887011528015, -0.6478651165962219, -0.8236644864082336, -1.8023065328598022, 0.9357530474662781, 0.5359717607498169, -2.194385528564453, -0.7170450091362, 0.02957775443792343, -0.016924748197197914, -1.1310497522354126, -0.039104435592889786, 0.9556571245193481, -1.1775732040405273, 0.9457593560218811, 0.3767072260379791, 0.27973273396492004, 1.173090934753418, 0.21680594980716705, 0.612924337387085, -0.5144392251968384, 1.000442624092102, 0.3554537296295166, -1.3398622274398804, 0.02462380938231945, 0.5585426092147827, -0.585852861404419, 0.3282444477081299, 0.7763580083847046, -0.39893656969070435, 0.24767622351646423, -0.7062408924102783, 1.3333126306533813, 0.1335754543542862, -0.7687544226646423, 0.3145882189273834, 0.2992130219936371, 0.8273759484291077, -1.6095637083053589, 1.6172738075256348, -2.5624630451202393, -3.3696541786193848, -0.1550740748643875, -0.8192278742790222, -1.244564175605774, 0.7721660137176514, -0.7805843949317932, -0.41202718019485474, 0.4763827621936798, -0.5608438849449158, -0.07934913784265518, 0.8222856521606445, -0.9641782641410828, 0.5692228078842163, 1.7592114210128784, 1.221925973892212, -0.2950810492038727, 1.269262671470642, 0.8813818097114563, 1.145391821861267, 0.30167052149772644, 1.3148417472839355, 0.24873019754886627, -1.3926366567611694, -0.24402396380901337, 0.9923849105834961, -0.8062727451324463, 1.0904353857040405, -2.095533609390259, 0.9322154521942139, 0.11991298198699951, -0.8216806054115295, 2.591998815536499, 0.351189523935318, 0.8668246865272522, 1.1970857381820679, 2.173311471939087, -1.3692922592163086, 0.8709970116615295, 0.5023108720779419, 2.570319890975952, 0.1746366173028946, -0.3027009963989258, -0.8378618955612183, 0.46095478534698486, 1.5161939859390259, 1.0091487169265747, 0.9596680998802185, 1.1320477724075317, 0.03992069512605667, -0.39034461975097656, -1.5131765604019165, 1.2536205053329468, -0.10745308548212051, -2.5947821140289307, -0.34055227041244507, -1.1778641939163208, 1.680064082145691, -0.6790658235549927, 1.4881699085235596, 1.517349123954773, -0.5588741302490234, -0.5446581244468689, 0.0035645603202283382, 0.03173185884952545, -0.2934824824333191, -0.5138590931892395, -0.4111778140068054, -2.62323260307312, -1.2043999433517456, -0.34945130348205566, -0.06625493615865707, -0.25479891896247864, 0.7392596006393433, 0.5343311429023743, -0.3682486414909363, 0.3782489597797394, 2.2021799087524414, 0.9496263861656189, -0.42935654520988464, -0.4690735936164856, 1.1871893405914307, -0.6773525476455688, -0.9369813203811646, -0.12696650624275208, -1.3911124467849731, -0.7526956796646118, 0.726894199848175, -1.0502958297729492, 0.01231094915419817, -2.7026329040527344, 0.028325606137514114, 0.16838620603084564, -0.404528945684433, -1.1721827983856201, 0.8572167754173279, 0.2793921232223511, 0.8465414643287659, 1.0069208145141602, 0.19164353609085083, 0.7543870806694031, 2.0011043548583984, 0.11412813514471054, -0.12264163792133331, 1.132449984550476, 0.6794425845146179, -0.9593403935432434, 0.6470091342926025, -1.8747459650039673, 1.3294570446014404, -0.34915831685066223, -0.35184505581855774, 0.4518928527832031, -1.073730230331421, -0.36243388056755066, 0.34425410628318787, 1.260859727859497, 1.633083462715149, -0.713724672794342, 0.3309512138366699, -1.8981378078460693, -1.164426326751709, -0.5127401351928711, -0.11271266639232635, -0.4711674451828003, 0.27505016326904297, 1.1035454273223877, -0.43937423825263977, 1.955058217048645, -0.3559858500957489, 1.3225460052490234, -2.7071192264556885, -0.43763554096221924, -0.6608431935310364, 0.11732862889766693, -2.0539662837982178, -0.9110720753669739, 0.6534852981567383, -0.6776129007339478, 0.6235578656196594, -1.8949329853057861, 0.07177357375621796, -0.047637782990932465, 1.838435173034668, -1.104435682296753, -0.49731484055519104, 0.6664541959762573, 0.5168073177337646, -1.019959568977356, -0.7915239334106445, -0.2790504992008209, 0.26037073135375977, -1.5032447576522827, 0.41494956612586975, 2.46820330619812, 0.4929973781108856, -0.8018167614936829, 0.4607159495353699, -0.2041337490081787, -0.794402003288269, 0.31149765849113464, 0.47021088004112244, -1.388615369796753, -0.27270883321762085, -1.192659616470337, -0.07576961815357208, 1.1984856128692627, 0.7085554599761963, 1.169356346130371, -0.028775986284017563, -0.45034515857696533, -0.17331339418888092, 1.0577508211135864, 0.45497027039527893, 0.8161460161209106, -1.1737347841262817, 0.3800402879714966, -0.38343510031700134, -0.8852211236953735, -0.543491542339325, 1.3784693479537964, -0.48031967878341675, -0.3038738965988159, -0.3486631214618683, -0.6430991888046265, 0.03797898441553116, 0.6702562570571899, -2.2939932346343994, 0.9978323578834534, -0.15980996191501617, 0.5750547647476196, 0.38198795914649963, -1.2413678169250488, -0.19993674755096436, 0.08450271934270859, -1.0645440816879272, 0.5235722064971924, 0.0789477676153183, -0.42727941274642944, 0.5497738718986511, 0.597727358341217, -0.11962353438138962, -0.40247389674186707, -0.4036649167537689, -0.16662180423736572, 0.7885437607765198, -0.8682759404182434, -1.0326604843139648, -1.6011775732040405, -0.14736595749855042, 0.01437135599553585, 0.19090090692043304, -0.07671947777271271, -1.964412808418274, -1.2920260429382324, -0.7362580895423889, 0.32440370321273804, 0.2933613359928131, -0.1351093351840973, -1.0145182609558105, 1.4031044244766235, -0.8271932005882263, -0.7264096140861511, 2.3493478298187256, 0.03156643360853195, -0.40677696466445923, 1.6831932067871094, -0.23682095110416412, -0.10290583968162537, 1.7070388793945312, -1.1411341428756714, -0.37092217803001404, -0.3627868592739105, 2.6209065914154053, 0.26607730984687805, 0.8009966611862183, 0.760715901851654, -0.3131648600101471, -1.5722851753234863, -0.683445394039154, -1.4567465782165527, 2.029254198074341, -0.3396996557712555, 1.2667250633239746, 0.5609294176101685, -0.2552785873413086, -0.36509615182876587, 1.5602805614471436, -1.0001407861709595, 0.19062553346157074, 0.9362952709197998, -1.8866922855377197, 0.4077633321285248, -0.7316771745681763, -0.5061376690864563, -0.09765007346868515, 0.7425392270088196, 0.6028452515602112, -0.5793525576591492, 0.763126015663147, -2.6208019256591797, -2.5009727478027344, 0.5365344285964966, -0.22505123913288116, 0.8236535787582397, -0.7520629167556763, -1.2067596912384033, -0.09578090906143188, -1.4525256156921387, -0.5750902891159058, 0.6273534297943115, -0.016745077446103096, 0.4037765860557556, -1.6300524473190308, -0.7541630864143372, 0.3105982542037964, 0.36063334345817566, -0.009823627769947052, 1.0988825559616089, 0.5945262312889099, 0.1839628666639328, 0.36036086082458496, 0.16248367726802826, -0.42617425322532654, -1.7895442247390747, 0.46652358770370483, 0.7106462121009827, -1.3637621402740479, -1.2445369958877563, -0.29573968052864075, 0.27389097213745117, 0.2255653589963913, -0.14081737399101257, 0.5832534432411194, -0.16974423825740814, -1.9080169200897217, 0.254294753074646, 0.07649052143096924, -1.3328946828842163, -0.11282116919755936, 0.2508951723575592, 0.13058193027973175, 2.06722092628479, -0.7338043451309204, 1.2222392559051514, -0.3078814744949341, -1.198189616203308, 0.9676984548568726, 1.8732656240463257, 0.6172453165054321, 0.5872587561607361, 0.8006206750869751, 1.626202940940857, 1.1871058940887451, 0.5320218801498413, 0.07860229909420013, 1.1469752788543701, 0.009727134369313717, 2.2595746517181396, 0.7395470142364502, -0.39826905727386475, -0.9571064114570618, -1.3211565017700195, -1.6978727579116821, -1.1030174493789673, 1.465325117111206, 0.43776407837867737, 0.49289000034332275, -1.1650748252868652, -0.11792290210723877, -0.4057656526565552, 0.6340444087982178, 0.15941055119037628, -1.0227060317993164, 0.6650516986846924, 0.9183769822120667, 1.2931489944458008, -2.4585280418395996, -1.4821593761444092, -0.3889947235584259, -0.028746040537953377, -0.16238901019096375, -0.18487507104873657, 1.7569993734359741, -2.330071449279785, 1.7482056617736816, 2.2847421169281006, -1.2628031969070435, 0.8364963531494141, -0.47954049706459045, -0.2779557704925537, -0.9598592519760132, -0.2723383903503418, 0.006491669919341803, -1.125374674797058, -1.3104729652404785, -0.7489739060401917, 0.2866577208042145, 2.0165023803710938, -0.5528672337532043, -0.78031986951828, 0.7110437154769897, 0.08436915278434753, -1.4445182085037231, 1.1016106605529785, -0.5034974217414856, 1.2328319549560547, -0.06438937783241272, -0.4544468820095062, -1.256913661956787, 2.361790180206299, 0.6220779418945312, 2.2185323238372803, -2.1777403354644775, -0.7302411198616028, 0.13510462641716003, 0.8801645636558533, 0.6474248766899109, -0.5450482964515686, 2.058839797973633, -0.10983595252037048, 0.3850654661655426, -0.8117324113845825, 1.0826984643936157, 0.8435529470443726, -0.2066870629787445, -0.6787363290786743, -1.2250066995620728, 0.4429556727409363, -0.145328551530838, -0.4615073502063751, 0.3480805456638336, 2.1828107833862305, -0.20901204645633698, 0.052151333540678024, -0.038955919444561005, -1.409756064414978, 0.1392878144979477, 0.4863525927066803, -0.1556505560874939, 0.9456501603126526, 0.9060636162757874, 1.211693525314331, 0.7273733019828796, -1.8428826332092285, 1.574580430984497, -0.31486162543296814, -0.35480666160583496, -1.5500037670135498, 0.9135758280754089, -1.3842650651931763, 2.194063663482666, 0.304431676864624, -0.2325488030910492, 2.0302252769470215, 1.222826600074768, -0.3430600166320801, 1.440202236175537, 1.3145718574523926, -1.1528549194335938, 0.567611813545227, 0.8030272722244263, -0.42639535665512085, -0.23233246803283691, -1.1528615951538086, 1.2143441438674927, -1.1923840045928955, 1.196898341178894, -0.14272618293762207, 0.11563178151845932, -0.6288104057312012, 2.191969394683838, -0.07744099944829941, -0.41411545872688293, -0.14863410592079163, -1.5597155094146729, 0.7989278435707092, -2.308455467224121, -0.6755734086036682, -0.7695436477661133, 0.8455827236175537, 1.0675569772720337, 2.1623921394348145, 1.392468810081482, 0.49534764885902405, -0.7604480981826782, 0.548976719379425, 2.567845344543457, -0.5181111693382263, -0.6837804913520813, -1.5882713794708252, 0.37864506244659424, 1.27643620967865, -0.9428830146789551, 1.6345090866088867, 0.2950354516506195, -0.6008802056312561, -0.29327788949012756, -0.06940191239118576, -1.507582187652588, 1.5929524898529053, -0.09144474565982819, 0.29142892360687256, -0.6918917894363403, -0.5735728740692139, -0.9232064485549927, 0.7144014239311218, 0.051533304154872894, 0.7401901483535767, 2.56978440284729, -0.19396743178367615, 2.056569814682007, 1.478463888168335, 1.32172691822052, 0.37935158610343933, -0.5855051279067993, -1.053026556968689, 0.542953372001648, -0.4512978494167328, -1.443160891532898, 1.0273840427398682, 0.05133720859885216, -0.11877810209989548, 1.4746904373168945, -0.34056779742240906, 0.5903337001800537, 0.4747520685195923, -0.050466395914554596, 0.8588889241218567, -1.9207690954208374, -0.4550597667694092, -0.3772772252559662, -0.9123722910881042, -1.8703724145889282, 0.2430192530155182, 1.2036879062652588, -1.2914286851882935, -0.5735601782798767, 0.7460415959358215, 0.19409644603729248, 0.445837140083313, -2.3830230236053467, -0.05509592592716217, -0.029688524082303047, -0.8561134934425354, -1.247995138168335, 0.2832261621952057, 2.55660080909729, -0.9438773989677429, -0.04911842942237854, 0.07464973628520966, -0.2605554759502411, -0.009621890261769295, -0.39366453886032104, -0.31562456488609314, 0.6504176259040833, -0.5566336512565613, -0.9553090929985046, 1.4751402139663696, -0.20605242252349854, -0.04005051404237747, -0.5637929439544678, 0.5927843451499939, 1.4258275032043457, -1.8348692655563354, 1.0609662532806396, -0.8538082242012024, 0.08571886271238327, -0.10213949531316757, 0.4115986227989197, 1.244879961013794, -1.0631433725357056, 0.8765150308609009, -0.819359302520752, -0.8324199318885803, 0.2275431752204895, -0.20006178319454193, 0.2481233924627304, -0.551308810710907, -0.008618338964879513, 0.29494622349739075, 0.15150929987430573, -1.7009077072143555, -0.3791648745536804, -0.7732191681861877, -0.624734103679657, -1.268512487411499, -0.2690534293651581, -1.1393444538116455, 0.04952172935009003, -0.6421937346458435, -0.10954599827528, -0.3540009558200836, 0.4905015230178833, -1.6871639490127563, 0.008856282569468021, 0.6204884648323059, 0.27881988883018494, 0.29744037985801697, 0.6407867074012756, 1.6991264820098877, -1.8841826915740967, -0.06569533795118332, 0.5414841771125793, -0.18950390815734863, -0.2890245020389557, 0.340839684009552, -0.18341873586177826, -0.562468945980072, -0.8159339427947998, -0.9681617617607117, 1.6880271434783936, -0.573329508304596, -0.2840736508369446, 0.3982771933078766, -0.17426922917366028, -0.032047074288129807, 0.6870754361152649, -0.20270779728889465, 0.2816406786441803, -1.0200798511505127, 0.5542857050895691, -1.280080795288086, -0.16814759373664856, 0.1965874433517456, -1.062275767326355, 0.0980476662516594, 0.7628425359725952, -1.3606759309768677, 0.06286387145519257, 0.877770721912384, 0.46710577607154846, -0.872824490070343, 0.5448265075683594, 0.9509842991828918, -0.48015910387039185, 0.29300299286842346, 1.2903234958648682, -1.5155600309371948, 0.5133731365203857, -0.31393226981163025, -1.4068567752838135, -3.4995508193969727, 0.7377650737762451, -0.9654210209846497, 0.7801045775413513, -0.3942510187625885, 0.8956670761108398, 0.246390700340271, -0.2465120404958725, 0.4421168565750122, -0.09912899136543274, -0.6892821788787842, 0.9467089176177979, -2.0294532775878906, -0.2323998510837555, -1.377321720123291, 1.6151938438415527, 1.028363823890686, -0.1481388360261917, -1.5510280132293701, -0.7846812009811401, -1.953263282775879, 0.5603775382041931, 0.869147002696991, 2.1424715518951416, -0.4872764050960541, 0.9645618796348572, 0.404428631067276, -0.4838765859603882, -0.5738930702209473, -1.5964573621749878, -2.768829107284546, -0.25471559166908264, -0.14287878572940826, 1.068785548210144, 0.6277888417243958, -0.0975581705570221, -1.1492640972137451, 0.009126191027462482, -1.0723844766616821, 1.2027487754821777, -0.7504947185516357, -1.5253260135650635, -1.0719692707061768, -0.9255493879318237, 0.15315276384353638, -2.0532405376434326, 0.10154420137405396, -1.1563268899917603, -0.35409143567085266, 1.2321174144744873, 0.08549439907073975, -0.035003334283828735, -1.0911791324615479, 0.06828301399946213, -1.1843822002410889, 0.7591248154640198, 0.7592605352401733, -0.19352684915065765, -2.045135498046875, -0.5500879883766174, -0.4193914532661438, 0.16138289868831635, 0.5386321544647217, 1.7816230058670044, -0.6468935608863831, -0.37857893109321594, 0.8519774675369263, -1.3043056726455688, -1.0413012504577637, -0.5005406141281128, 0.9547396302223206, 1.1954745054244995, 0.3446200489997864, -0.08622017502784729, -0.3080635666847229, 0.3218235373497009, -1.708654522895813, 0.18831072747707367, -0.10203441232442856, -0.20159688591957092, 1.4725747108459473, -0.16188931465148926, 2.8421003818511963, 0.36085695028305054, -0.09678659588098526, -0.07604032754898071, -0.6525298953056335, -0.03770207613706589, 0.09087210148572922, 0.46923837065696716, 0.7898412942886353, 0.9412949085235596, -0.6220235228538513, 0.5068049430847168, 0.8054354190826416, 0.20488294959068298, 0.5298857688903809, -1.6939111948013306, 0.02156844548881054, -0.7215161919593811, 0.003518662415444851, -0.35452455282211304, 0.22481004893779755, -1.5482922792434692, 0.13794469833374023, -0.4278585612773895, -0.5462290644645691, -0.3860081136226654, 0.7730686664581299, -2.152432441711426, -0.022802947089076042, -1.4378949403762817, 0.05897116661071777, -1.0018845796585083, 0.811374306678772, -0.323988676071167, -0.11615920811891556, -1.0317703485488892, 0.12420102208852768, 1.0062288045883179, -0.678961992263794, 2.329627513885498, 0.182785302400589, -2.5592517852783203, 0.7295709252357483, -0.7303610444068909, 1.6659358739852905, 0.5429365634918213, 1.5520247220993042, -0.8749749064445496, 0.16481401026248932, 0.6763835549354553, 1.3850890398025513, -0.9569982886314392, -0.7732764482498169, -0.31969478726387024, -0.3506975471973419, 0.31151053309440613, 0.40744268894195557, 0.39174237847328186, 0.9035201668739319, 1.457196593284607, -0.41231340169906616, 1.4789704084396362, -1.3461815118789673, 0.1471439152956009, 0.3580099046230316, -0.8092142939567566, 1.591199278831482, -0.6170219779014587, -0.0627947598695755, -0.5572090744972229, 0.6695098280906677, 0.5046478509902954, -0.6737927198410034, -0.23438890278339386, 2.8687808513641357, 0.4822145700454712, 0.6977007389068604, -0.5078702569007874, 0.02266325242817402, -1.369300127029419, -1.7750738859176636, 1.077610731124878, -0.9298906922340393, 0.11858676373958588, -1.058284878730774, 1.7449918985366821, -1.0471020936965942, 0.5043119192123413, -0.7564653754234314, 0.45943769812583923, 0.05628399923443794, -1.0366675853729248, -0.3847046494483948, 0.324493408203125, -0.9732152223587036, -0.42828479409217834, -1.490053653717041, -0.2689040005207062, 0.8175351023674011, -0.5652023553848267, 1.4265793561935425, -0.9307888746261597, -0.42510855197906494, -1.1273231506347656, -2.3443682193756104, -0.3637281656265259, -0.6868511438369751, 0.9181003570556641, 0.21850299835205078, 2.7592976093292236, -0.36276155710220337, -0.9513252973556519, 0.06750507652759552, 1.2522519826889038, 1.0555435419082642, 0.29490405321121216, -0.3971377909183502, 1.018303394317627, 0.008271215483546257, -0.28448134660720825, 0.16008271276950836, 0.5890607833862305, -0.24070827662944794, 1.010159969329834, 0.36438247561454773, 0.0007312206435017288, 0.8399525284767151, 0.4159325063228607, -0.16135472059249878, 1.557224154472351, 0.44956302642822266, 2.140146493911743, -0.49475231766700745, -1.097475290298462, -0.6728222370147705, 0.466369092464447, 0.06490196287631989, -0.8030748963356018, -0.593221127986908, -0.5017610192298889, -0.33298078179359436, 0.327490895986557, -0.08662586659193039, 0.4934631288051605, -0.9442703127861023, 0.6928496360778809, 0.8201981782913208, 0.1259126365184784, 0.02505004033446312, 1.1324690580368042, 0.6063265204429626, 0.379537433385849, 1.0097532272338867, -1.2386887073516846, 0.788832426071167, 0.8785626292228699, 1.266850233078003, 2.127643585205078, 0.3123728930950165, -0.7103789448738098, -0.2339131385087967, 0.5318379402160645, 0.07100632041692734, 0.06724970787763596, 0.08921632915735245, -1.1596226692199707, -0.7873283624649048, 0.9023197293281555, 2.295992374420166, 0.7444953322410583, -1.4802525043487549, -0.3807036578655243, 0.5268284678459167, -0.9734468460083008, 1.2778382301330566, -0.3071921467781067, -2.178192615509033, 0.11284715682268143, 0.3247472941875458, 0.8393868207931519, 0.0767185240983963, -0.2151959240436554, -1.609897494316101, -0.5954874753952026, 0.6438161730766296, 0.11916995793581009, -0.9654342532157898, 0.007853973656892776, -0.3387305438518524, -1.266918420791626, -0.9519868493080139, -1.2185243368148804, -0.24046838283538818, -0.5673398375511169, 1.0549522638320923, 1.0552064180374146, 1.023378849029541, -0.038182277232408524, 0.6616647243499756, 0.6452085375785828, -0.9901250004768372, 2.051809072494507, 0.6926209926605225, 1.304087519645691, -0.5610249042510986, -0.545462429523468, -1.7389397621154785, 0.787074863910675, -0.3833668828010559, -1.044567346572876, -0.10860278457403183, 0.7576969861984253, 0.08045904338359833, 0.9106532335281372, 1.6117035150527954, 0.622321605682373, 1.0547542572021484, 0.67746901512146, -0.7374098896980286, -0.08514435589313507, 0.37534719705581665, -0.8702115416526794, -0.6833168268203735, 0.38787469267845154, 1.0348918437957764, -0.8179505467414856, 1.0958157777786255, -0.06306757777929306, -0.7615209817886353, 0.06256023049354553, 0.2184954434633255, -1.1563578844070435, -0.943015992641449, 0.5795185565948486, 0.20367075502872467, -1.3562862873077393, -1.4864979982376099, -2.0458481311798096, -0.14351263642311096, 0.5226635932922363, -0.6254938840866089, -0.26090750098228455, 0.1937108188867569, 0.8795711398124695, 1.3243403434753418, 0.5924682021141052, 2.180457353591919, 1.620568037033081, 1.7539739608764648, -1.5023698806762695, -0.039639912545681, 0.1954474151134491, 0.9299131631851196, -0.1801321804523468, -2.2358052730560303, -1.1757577657699585, 2.1543972492218018, 0.5470017790794373, -1.3908172845840454, -0.49939751625061035, -0.13197116553783417, -1.2228492498397827, 0.44539424777030945, 1.214662790298462, -1.01875901222229, -1.8694061040878296, -0.7089757323265076, -0.3334778845310211, -1.3554775714874268, -0.4574194550514221, -0.7959914803504944, 0.018382318317890167, 0.4006015360355377, 1.6833164691925049, -1.4981781244277954, 0.4746446907520294, 0.48578545451164246, 0.4162740707397461, 1.7920111417770386, -0.2212820202112198, -1.23312509059906, 1.2871934175491333, 0.7229412794113159, 0.41054660081863403, 0.6244369745254517, 0.040655795484781265, -1.2765958309173584, -0.3137800693511963, 3.0918869972229004, -1.2810300588607788, -0.2515995502471924, -1.3466506004333496, -0.296945184469223, 1.0413116216659546, -0.28011369705200195, 1.7076259851455688, -1.6106983423233032, 0.5294298529624939, 1.2226316928863525, -0.7871044874191284, 0.45052868127822876, -1.8148610591888428, 0.49693167209625244, 0.5303730964660645, -1.8943560123443604, -0.5831460952758789, -0.21868664026260376, 1.2285218238830566, 2.5266292095184326, -0.9471998810768127, 1.5115374326705933, -2.355252981185913, 1.5124058723449707, -0.2395734041929245, 0.17399580776691437, -0.11650168150663376, 0.5642609000205994, -1.0942081212997437, 1.0086400508880615, -0.23595954477787018, 1.0161832571029663, -0.4147227108478546, 1.1561354398727417, -1.951724648475647, -0.011941681616008282, -1.5085666179656982, -1.1410621404647827, 0.9082664847373962, 1.6500827074050903, -0.8402462005615234, 1.221411108970642, -0.011470909230411053, -0.9167001843452454, 0.32044222950935364, 2.6684701442718506, -0.08164473623037338, -0.9733769297599792, -0.7798989415168762, 1.297793984413147, 1.1350525617599487, 0.3664298951625824, 0.45566263794898987, 1.2134627103805542, -0.03532399237155914, -0.42822566628456116, 1.342434287071228, 1.6892526149749756, -0.12565013766288757, -0.47925707697868347, -0.46000936627388, -0.8518704771995544, -0.5513461232185364, 1.0885483026504517, -0.2155100405216217, 0.46254321932792664, 1.1576088666915894, -0.6094764471054077, -0.7879744172096252, -0.7509159445762634, 0.3818129003047943, 1.177822232246399, 0.5664801001548767, 1.2273292541503906, 0.48551759123802185, -1.453243613243103, -0.41415897011756897, 0.8615351915359497, -0.6955032348632812, -0.27566635608673096, -0.06874848902225494, 1.2451390027999878, -0.38241347670555115, 0.3208058476448059, -0.46620112657546997, -1.8063163757324219, -0.07408007234334946, -1.224057674407959, -0.8682390451431274, -1.3338946104049683, 0.04149336367845535, 0.9949064254760742, 2.105703353881836, 1.60672128200531, -0.9378330111503601, 0.5551780462265015, 0.8093047142028809, 1.6703741550445557, 0.6918991208076477, 0.6062118411064148, -1.052864909172058, 0.858841061592102, -0.1744404435157776, 1.0939360857009888, -1.2168718576431274, 0.919455349445343, -0.6245653629302979, 0.22285421192646027, 0.2113185077905655, -0.7526684403419495, -0.41195547580718994, 0.7127153873443604, 0.5148829817771912, -0.5728989839553833, 1.506393551826477, 1.0912630558013916, 1.130973219871521, 0.1758333444595337, 1.577433466911316, 0.10271348059177399, 0.41628706455230713, 0.32884231209754944, -1.0910284519195557, -0.6810480952262878, -0.29461926221847534, 0.427649587392807, -0.5820872187614441, 0.5839722752571106, 1.1703993082046509, 1.4940663576126099, 0.7888424396514893, 0.48996493220329285, 0.627804696559906, -0.2891721725463867, -0.3812193274497986, -0.754464328289032, -0.41103750467300415, -0.33001452684402466, 0.09716245532035828, 1.2186238765716553, 0.7848864197731018, 0.3606616258621216, -0.809581995010376, 1.3853917121887207, -1.4347659349441528, -0.9472021460533142, 1.6988011598587036, 2.210118532180786, -0.6301257610321045, 1.693307638168335, 1.2268046140670776, 0.48938077688217163, 1.0523011684417725, 1.752219557762146, -1.6613142490386963, -0.9973418712615967, 0.303886741399765, 0.36118295788764954, 0.3805336654186249, 0.9916043281555176, 0.9228052496910095, 1.7144520282745361, -0.04221628978848457, -0.20024824142456055, 1.2169381380081177, 0.7759560346603394, -0.8580347895622253, 0.36924293637275696, -1.6891038417816162, 0.8187082409858704, -1.2797893285751343, 0.5502417087554932, 0.7027474045753479, -0.4072694778442383, 0.4440214931964874, 1.223214864730835, -0.293845534324646, 0.85865318775177, 0.28714191913604736, 1.3094677925109863, 0.31148383021354675, 1.7113282680511475, 1.6846827268600464, 0.19134168326854706, 1.409720540046692, -0.5602556467056274, 0.2733146846294403, 0.33675312995910645, -0.4310669004917145, -0.5353068709373474, 0.41884955763816833, 1.003578782081604, 0.02807454578578472, -1.786801815032959, -2.3775243759155273, -1.3046435117721558, 1.3738785982131958, -1.7121553421020508, 1.8406585454940796, -0.07604877650737762, 1.443725824356079, 1.66862952709198, 1.0335391759872437, 0.08642882853746414, 0.46084290742874146, 0.609143078327179, 0.013721984811127186, 0.006145329214632511, -0.8551350235939026, 0.93137127161026, 0.7723921537399292, 0.3186066150665283, -1.0765986442565918, 0.5105425119400024, 0.43703964352607727, 0.42264634370803833, -1.614114761352539, -2.150290012359619, -1.1753566265106201, 0.6849023103713989, -1.1405460834503174, -1.6475486755371094, 2.8191401958465576, 2.032759428024292, 1.0495355129241943, 0.3276415169239044, 0.9834493398666382, 0.32788926362991333, 0.5104051232337952, 0.8992362022399902, -0.3875962495803833, -0.7033525705337524, 1.5417035818099976, 0.8469973802566528, 0.716438889503479, -1.256806492805481, 0.7654590010643005, 0.49396267533302307, 0.1483292281627655, -0.47441622614860535, -0.3275487422943115, 0.5041687488555908, 1.1281379461288452, 0.7095222473144531, 0.5803142786026001, 0.2719651460647583, 2.238084554672241, 0.36367732286453247, 0.5376766920089722, 1.3203333616256714, 2.682798385620117, 1.1543610095977783, 0.24741877615451813, 1.0040868520736694, -1.3067620992660522, 0.10659436136484146, -2.25212025642395, 1.1021589040756226, 0.6806278228759766, 0.1306672990322113, 0.5788754820823669, 0.6155888438224792, -1.6075342893600464, -1.6931657791137695, 0.8971264958381653, -1.1248306035995483, -0.8962514400482178, 0.8194222450256348, -0.2959558367729187, -0.6506388783454895, -1.4736028909683228, -2.4263293743133545, -0.3031889796257019, 0.2771082818508148, -0.7363154292106628, -0.2086016982793808, 0.21747379004955292, -0.7550818920135498, -1.256095051765442, -0.03371400758624077, 0.7023900747299194, 0.16036102175712585, -1.1748943328857422, -0.9250844120979309, -1.4182759523391724, 0.6199857592582703, -1.219645619392395, 0.5618604421615601, 1.1805509328842163, -0.6128658652305603, -1.3095896244049072, -0.7020530104637146, -0.5390225052833557, -0.1930498480796814, -0.09900879859924316, -0.23101727664470673, 1.9592174291610718, -1.5500688552856445, -1.0020074844360352, 2.0300989151000977, 0.29536861181259155, -0.11667802184820175, -0.22296270728111267, -0.5632572174072266, -0.12952013313770294, -0.76231449842453, -0.8327909708023071, -0.7151422500610352, 0.9330930113792419, 0.2020140141248703, -1.3337219953536987, -0.6666591167449951, -0.45561373233795166, -0.3398422598838806, 0.8611280918121338, 1.024314045906067, -0.7509826421737671, 0.1463109403848648, -0.47084134817123413, -0.9033157229423523, 0.8210179209709167, -1.3378218412399292, -0.9105680584907532, -0.7147942185401917, 0.13676817715168, -0.7575318217277527, -0.3435543477535248, -0.7134798765182495, -1.3433575630187988, 1.0712313652038574, -0.24660439789295197, -0.1663249433040619, -2.4935925006866455, 0.7996471524238586, 1.296249508857727, 0.5390326976776123, 1.171697974205017, 0.806126058101654, -0.9791902899742126, 0.737190842628479, 0.29202884435653687, 1.7062773704528809, 0.11000331491231918, 0.47350531816482544, -0.7309873104095459, 1.1883183717727661, 0.10325443744659424, 1.837007999420166, -0.502129852771759, 0.04056205227971077, -1.6515458822250366, -1.238028883934021, -0.06313981115818024, 0.8089039921760559, 0.5785384178161621, -1.0555469989776611, -0.7695601582527161, -1.0912569761276245, 0.7149233222007751, -0.3497348129749298, 1.0576248168945312, -0.45747140049934387, 0.7377479672431946, 1.1110670566558838, 0.6862483620643616, 0.33977994322776794, -0.8208857774734497, -0.05702028423547745, 0.45598340034484863, -2.406998634338379, 0.9778606295585632, 0.9694875478744507, -0.03673204407095909, -0.1668352335691452, -0.7677940726280212, -0.4755761921405792, 0.016612350940704346, 0.2129596620798111, 0.7659022808074951, -0.03867129981517792, 1.857495903968811, 0.7415507435798645, -1.0094432830810547, -1.1696720123291016, 0.1424700766801834, -1.5923314094543457, -0.2640433609485626, -0.3183016777038574, -0.2851475775241852, 0.05437365174293518, 1.050094485282898, -1.0418930053710938, -2.121408224105835, -1.0969127416610718, -2.146481990814209, 0.37947800755500793, -1.0725526809692383, 0.2697124481201172, 0.9063783288002014, 0.7924642562866211, -1.5399481058120728, 0.15013891458511353, 2.979241132736206, -2.0014936923980713, 2.0184810161590576, 0.7861029505729675, 1.5248957872390747, 1.0181385278701782, -1.8016480207443237, 2.8701224327087402, 0.1830378919839859, 0.009157453663647175, 1.1422877311706543, -0.29761824011802673, 0.6003756523132324, -1.4028927087783813, 1.5638248920440674, -0.3850601315498352, 0.5168285965919495, -1.1740872859954834, 1.0162594318389893, -0.25010693073272705, -0.21766746044158936, -1.515833854675293, 0.18486739695072174, -0.5409709811210632, -1.028655767440796, 0.06926793605089188, 1.5151572227478027, 0.2553253471851349, 0.09168564528226852, 0.2739100456237793, -0.7322324514389038, 0.9581056237220764, -1.484351634979248, 0.9880775809288025, -0.6462902426719666, 0.5173008441925049, -1.5956315994262695, -0.10462504625320435, 0.9696666598320007, -0.03260647878050804, -0.3612404465675354, -0.7437014579772949, -0.8685764670372009, 0.3722821772098541, -0.03433206304907799, 0.9054279327392578, 0.25498923659324646, 0.5181610584259033, 1.267999529838562, 1.0862693786621094, 1.0967810153961182, -0.20656587183475494, 0.8300164937973022, 0.1668967604637146, 0.48216113448143005, -1.217827558517456, 0.5570747256278992, 0.37347257137298584, 0.1545991450548172, 0.20624004304409027, 2.929121732711792, -0.3824477195739746, 0.23880308866500854, 1.6469974517822266, 1.7639955282211304, 0.6894791126251221, 0.7429144978523254, -0.5475508570671082, 0.9008873105049133, 1.3346294164657593, 0.13471730053424835, 0.8441727757453918, -1.1588481664657593, -0.766431450843811, -0.293605774641037, 1.4320193529129028, -0.11539384722709656, 0.25223931670188904, -1.0157779455184937, 0.5453954339027405, -1.2438589334487915, -0.6067362427711487, 0.5197631120681763, 0.22327031195163727, 1.5124552249908447, -0.21980896592140198, -0.8979771733283997, 0.09556513279676437, -0.08910583704710007, -2.4588680267333984, -1.2426894903182983, 0.831180989742279, 0.07891029864549637, -1.779343843460083, -0.2477693259716034, -1.907684326171875, 1.1316161155700684, -0.45384669303894043, 0.6632290482521057, 0.6179429292678833, -2.9822897911071777, -0.6914950013160706, 1.7677339315414429, 1.4111826419830322, -0.2918219566345215, -0.07437052577733994, -2.0160720348358154, -1.2013516426086426, 1.4681614637374878, -0.37176498770713806, -0.36729922890663147, -0.7157317399978638, -0.2240152657032013, -0.402243435382843, -2.875331401824951, -0.3758802115917206, -0.5342352390289307, 0.32701173424720764, -0.1291186660528183, 0.07994641363620758, -0.13345444202423096, -1.7385807037353516, 0.40252885222435, -0.04828297346830368, 0.5128881931304932, -0.7698389291763306, -0.4370608627796173, 0.09598974883556366, -1.490476131439209, -0.09072883427143097, 1.3915337324142456, 0.16593891382217407, -0.2463529109954834, -1.6977664232254028, 0.502601683139801, -0.637067973613739, 0.100483238697052, -1.0716081857681274, -0.19683057069778442, -0.3742082715034485, 0.20680053532123566, -1.1639235019683838, 0.42707324028015137, -0.7397201657295227, 0.7518007755279541, 0.1641918420791626, 0.4939131438732147, 0.08681642264127731, -0.36659446358680725, -1.8582651615142822, 0.04926268756389618, 0.46667250990867615, 2.260632038116455, 0.5524598956108093, 1.1215062141418457, -0.03290775045752525, -1.9868494272232056, 1.1119494438171387, 0.05660402029752731, -0.006920395419001579, 0.42526209354400635, 1.87494957447052, 0.5292790532112122, 0.8355700373649597, 0.22152988612651825, 1.157779335975647, -0.7115678787231445, 0.6669341325759888, 0.25324684381484985, 0.24136781692504883, -1.065068244934082, -0.1789655089378357, -0.8108016848564148, -0.49318385124206543, -0.0011300561018288136, 0.7435081005096436, -0.7749835848808289, 1.5089576244354248, -1.4851157665252686, -0.6320840120315552, -1.2053039073944092, -0.26823028922080994, 0.7458070516586304, -0.39694398641586304, -1.5341976881027222, -0.5046027898788452, -1.3952977657318115, -0.12097818404436111, -0.25339919328689575, 0.07791390269994736, -1.5075210332870483, -0.09811938554048538, 0.05424240231513977, -1.1461730003356934, 0.6139971613883972, 0.08230552822351456, 1.5507488250732422, 0.3237145245075226, 0.26721253991127014, 0.7206739783287048, 1.291098713874817, -1.4821867942810059, -0.17275470495224, -1.0631564855575562, 0.5033499598503113, -0.6702376008033752, -1.4465723037719727, -0.5040571093559265, -0.6162071824073792, 0.6175498366355896, -1.459973692893982, -1.2826213836669922, -0.9361194968223572, -1.3754072189331055, -0.3742513358592987, 0.5826190710067749, 0.369749516248703, -1.2446303367614746, 2.046605110168457, -0.8796321749687195, 0.9424861073493958, -0.1387692391872406, -1.0572900772094727, 0.027994774281978607, 0.18111786246299744, -0.12573377788066864, -1.432151436805725, -0.33531880378723145, -0.5873640179634094, -0.09012117981910706, -0.6094955205917358, 1.8228353261947632, -0.741503894329071, 0.36930394172668457, -1.0010441541671753, 1.1687235832214355, -0.3754737675189972, 0.5737752914428711, 0.4378921091556549, 0.7081468105316162, 3.164623260498047, -0.0420784130692482, 0.25620096921920776, -2.713581085205078, 2.2193093299865723, 0.6081618666648865, 0.4490930438041687, 0.20134681463241577, -0.10959228873252869, 0.22694231569766998, -0.8064393401145935, 2.121034622192383, 1.806734323501587, 0.004522593691945076, 0.9969769716262817, 1.0266878604888916, -0.2930634021759033, -2.186876058578491, -1.627236247062683, -0.8964093923568726, -1.0664093494415283, -0.6170181632041931, 0.8536491990089417, -2.3600950241088867, 0.007331118453294039, 0.8547085523605347, 1.1755794286727905, 0.4603208303451538, 1.638797402381897, 0.15848883986473083, -1.530666708946228, -0.5099679231643677, 0.6494866609573364, -0.08747963607311249, 1.154543399810791, 0.10971558094024658, 0.9274017214775085, 1.646531581878662, 0.7725321650505066, 0.6192336082458496, 0.584661602973938, -1.0979567766189575, 1.5104411840438843, 0.554370105266571, 1.827172875404358, 1.2882053852081299, -0.5904155373573303, -0.9650263786315918, -0.2979574501514435, 0.9872625470161438, -0.7812800407409668, -0.7713645696640015, 0.809467613697052, -0.016622673720121384, 2.2491908073425293, 0.26492416858673096, 0.21065235137939453, -0.7061442136764526, 2.4582784175872803, -0.7427368760108948, -0.7087076902389526, -1.6163073778152466, -0.5437079071998596, -0.06232528015971184, -0.5905623435974121, -0.39323899149894714, -1.0141735076904297, -1.0157105922698975, -0.1297747939825058, 1.105449914932251, -0.46010592579841614, 0.10713913291692734, 0.08771740645170212, -1.1938912868499756, -0.7661879658699036, 2.407282590866089, -1.1164673566818237, -0.25814855098724365, -1.7438066005706787, 0.830281138420105, 0.3406846821308136, -0.21591268479824066, -0.02708788961172104, -0.15632222592830658, 1.068735122680664, 0.7586230039596558, 2.0739386081695557, -0.4970669150352478, 0.814307451248169, 0.6413322687149048, 0.4985755681991577, -1.8420664072036743, 0.8791975975036621, -1.746800184249878, 1.0013127326965332, -0.27578839659690857, 0.5844805836677551, -0.4623171091079712, 0.4935707151889801, -0.9456968307495117, 0.5694421529769897, 0.9729424118995667, -1.4008125066757202, 0.981559693813324, -0.407406210899353, 1.0410958528518677, -0.567561686038971, 1.0982874631881714, -0.4077270030975342, -0.023894380778074265, 0.1823592185974121, -0.738471508026123, 0.5388438105583191, -1.3987112045288086, 1.8598074913024902, -1.4801729917526245, 0.3288027048110962, 0.553063690662384, -1.1520918607711792, -0.18422777950763702, -0.6680145263671875, -0.7361827492713928, -0.639548122882843, 1.355027437210083, 0.46556487679481506, -0.44518864154815674, -0.08622080087661743, 1.469120979309082, 0.6121445298194885, -0.3349291682243347, -0.42633765935897827, 1.447313904762268, 0.9793969988822937, 2.299180746078491, -0.05790955200791359, 0.13878567516803741, -0.022026224061846733, 0.3885825574398041, -1.3949682712554932, 0.7042062878608704, 2.093607187271118, 1.9674264192581177, -0.5390877723693848, 2.3377175331115723, -0.6928572058677673, 0.764323890209198, -0.23473267257213593, -0.3886392116546631, 0.9482172727584839, -0.44863900542259216, 0.6774762272834778, -0.9926565289497375, -0.3601466715335846, 0.7753323912620544, -0.13465289771556854, 0.19017177820205688, 0.1909056305885315, -0.1655701845884323, -0.08860635757446289, -0.8189861178398132, -0.06463714689016342, -0.26630380749702454, 0.330757737159729, 0.9543185830116272, 0.101621612906456, -1.1274187564849854, 0.9398325681686401, 0.003649470629170537, 0.31428393721580505, 0.861454427242279, 0.44630318880081177, 0.7094656825065613, 1.3449161052703857, -0.21611547470092773, -0.1529323309659958, 0.5196197032928467, -0.7892718315124512, 0.05258890986442566, -0.3275309205055237, -0.20379842817783356, -0.03475126996636391, -1.7955082654953003, -1.0583535432815552, -2.051455020904541, 1.0790858268737793, -0.0636700764298439, -0.848222553730011, -3.496739625930786, -1.1007195711135864, 0.9651269912719727, 0.8978354334831238, -0.498351514339447, 1.3700532913208008, 0.5789014101028442, -0.44914233684539795, 0.2116490602493286, -1.0796469449996948, 0.3276960849761963, -1.5556206703186035, -0.6394530534744263, 1.4467597007751465, -2.501835823059082, 1.4158554077148438, -0.01491871103644371, 0.611013650894165, -1.0110230445861816, -0.29348883032798767, 1.23183274269104], [0.6426116228103638, 1.1297410726547241, 0.8096969723701477, -1.0443434715270996, 1.39599609375, -0.5077913403511047, -0.4102969467639923, 0.3476918041706085, -0.18033644556999207, -1.1725225448608398, -0.07489480078220367, 0.6097617149353027, 0.43656739592552185, -0.6956114172935486, -0.8520781397819519, 1.0442215204238892, 0.06768318265676498, -1.1690621376037598, -1.6745845079421997, -0.060906801372766495, 0.16210325062274933, 0.9262595176696777, 0.06167727708816528, -1.741466760635376, -0.11925603449344635, 0.42512354254722595, -1.3034679889678955, -0.4795815050601959, -0.9136903882026672, 0.7671005725860596, 1.3295187950134277, 0.3804372549057007, 0.02093561924993992, 1.2981055974960327, -1.362383246421814, -2.5796191692352295, 1.16836416721344, 0.027414465323090553, 0.05045581981539726, -1.9154715538024902, -0.8627326488494873, 1.0821058750152588, 0.3704988956451416, 1.2444300651550293, -0.44097205996513367, 0.30574485659599304, 0.474161297082901, 0.4137551486492157, -0.32071730494499207, -0.22547218203544617, -1.1511720418930054, -0.20025625824928284, 1.467349886894226, 1.8299050331115723, 1.6405024528503418, -0.6818820238113403, 0.4150971472263336, -0.4940364956855774, 0.25740137696266174, -1.144774317741394, 2.2818918228149414, 1.4285825490951538, -0.16096682846546173, 0.22418513894081116, -0.16336600482463837, 0.04640096798539162, 0.3099956512451172, 0.5840884447097778, 1.64247727394104, 0.8215161561965942, -1.5823965072631836, -1.6078534126281738, -0.01630600541830063, -0.8691930174827576, 0.9144094586372375, 1.287498116493225, 0.5803655982017517, 1.6476489305496216, -0.38086602091789246, -0.5134628415107727, -1.760048270225525, -0.43875688314437866, 1.2758485078811646, -0.5736410617828369, -1.0975764989852905, 1.5816679000854492, -0.21195079386234283, -1.2822952270507812, 0.5018250942230225, 0.8900195360183716, -1.0951210260391235, 1.1400495767593384, -0.2183942049741745, -0.8395190834999084, -1.1383246183395386, -1.011283278465271, -0.82220858335495, 1.615820288658142, 2.2224912643432617, -1.4356720447540283, 2.7027108669281006, -1.8435516357421875, -0.12157738953828812, 1.2660496234893799, -0.14065858721733093, -0.9836651086807251, -0.7420697808265686, -1.6343951225280762, 0.4544079601764679, -0.5536975264549255, 0.9011955261230469, -1.7399675846099854, -0.42111971974372864, 0.2080264538526535, 1.0968081951141357, 0.688113272190094, 0.5885326862335205, 0.7254340648651123, -1.3029074668884277, -0.7084510922431946, -0.9941504597663879, -0.6947202086448669, 0.6252886056900024, 0.6652932167053223, 0.6721571683883667, 1.7669341564178467, -0.9829652309417725, 0.11021027714014053, -0.5016642808914185, 0.34625282883644104, 0.9579797387123108, 0.1279553920030594, -0.15899671614170074, 0.4067438840866089, -0.19081644713878632, -0.5786523818969727, -0.5762817859649658, 1.0370582342147827, -0.5007773041725159, -0.4715922772884369, 0.6565234065055847, -1.762618064880371, 0.2435467541217804, -1.0776993036270142, 1.2367324829101562, -0.67680823802948, 0.492585152387619, -0.9454301595687866, 0.25736671686172485, 1.2073661088943481, -0.39819103479385376, -2.031229257583618, 0.9439728260040283, -0.5093928575515747, 0.14378079771995544, 0.14549334347248077, -1.6736193895339966, 0.049301877617836, 0.29021546244621277, 0.17454646527767181, 0.6687688231468201, -0.36427345871925354, 0.47417721152305603, 0.6816691160202026, -0.7456821799278259, 0.14543402194976807, -0.9888933300971985, 1.5190316438674927, -0.41462111473083496, 1.8848562240600586, -0.09421364217996597, 1.1588575839996338, -0.9733827114105225, -0.8270201683044434, 1.5428072214126587, -0.43308666348457336, 0.15189479291439056, 0.9949327111244202, 0.04870148375630379, -1.36822509765625, -0.8687079548835754, -2.074228048324585, 1.253617525100708, 0.3130798041820526, 0.6068499088287354, -0.6975377202033997, 0.38146543502807617, 0.6144384741783142, -0.562207818031311, 0.1637178510427475, -0.3791807293891907, -0.2920031249523163, -2.0926430225372314, -0.09610657393932343, -0.4817756414413452, 0.3126504123210907, -0.5537451505661011, -0.9879850149154663, -1.40598726272583, 1.0781618356704712, 1.3709582090377808, -0.3709114193916321, 0.4873116910457611, -0.7690775394439697, -0.3709120452404022, 1.3806610107421875, 0.9310237169265747, -1.131338119506836, 1.3147027492523193, -0.413173109292984, -0.20013803243637085, 1.9117958545684814, 0.8248516321182251, -0.7959629893302917, -1.4891712665557861, -0.925104022026062, 0.8011273741722107, 1.093645453453064, -0.11584985256195068, -0.6586158275604248, 0.26344725489616394, -1.6600948572158813, 0.8048877716064453, 0.8445559144020081, 0.5704486966133118, -0.15404832363128662, 1.0865936279296875, 0.22110292315483093, -1.182804822921753, 0.975256085395813, -1.8746130466461182, 0.8151569366455078, -1.0084147453308105, -0.0013191907200962305, 0.34705686569213867, 0.1704319566488266, 0.031152354553341866, -1.4298439025878906, 0.9349915385246277, -1.8677167892456055, 0.5249056816101074, -0.129958376288414, -0.46870169043540955, 0.611131489276886, -0.11428727209568024, 1.779043436050415, -0.19024285674095154, -0.3641151487827301, -0.5056746602058411, 0.5283396244049072, -0.9192965030670166, -0.50555419921875, 0.668808102607727, 1.4738686084747314, 1.380256175994873, 0.03166613727807999, -1.464720606803894, 0.17958442866802216, -0.5594843626022339, 0.5288706421852112, -1.2934156656265259, -0.7149851322174072, 0.4086938500404358, 1.0792888402938843, 1.0295904874801636, -0.34735819697380066, 1.3473381996154785, -0.27860113978385925, 1.465185523033142, 0.18749158084392548, 1.9941469430923462, 0.17634916305541992, 1.7869101762771606, -1.8798989057540894, 0.06386147439479828, 0.8398619890213013, -0.765636682510376, -0.5177478194236755, -0.3301073908805847, -1.7148799896240234, 0.6662822961807251, -0.24059058725833893, 1.6786164045333862, 0.987089991569519, 0.26374825835227966, -0.07010466605424881, -1.4036544561386108, -0.33729469776153564, 0.20412074029445648, 0.41452962160110474, -0.4031282663345337, -1.6234062910079956, -0.07597284764051437, 1.5111116170883179, 0.29031267762184143, -0.7471922636032104, 0.9490533471107483, 0.34846654534339905, 1.2382627725601196, -0.8966127634048462, 0.034195464104413986, 0.32216987013816833, -1.3364222049713135, -0.10389113426208496, -0.13791371881961823, -1.0890522003173828, 1.615325689315796, -0.3137093186378479, -0.3337218165397644, 0.587741494178772, -0.10475566983222961, 0.4680171310901642, -0.8925409317016602, -1.5732725858688354, -0.618928074836731, -0.7701961398124695, 0.5603052973747253, -0.5757522583007812, 1.2317287921905518, 1.0685656070709229, -0.2677481472492218, 0.26003044843673706, 1.6487700939178467, 0.4972214102745056, -0.46869155764579773, -0.442645400762558, 1.5546199083328247, 1.5964031219482422, -0.5444139838218689, 0.1765376329421997, -0.13695479929447174, -0.6718408465385437, 1.5292922258377075, -0.5795369744300842, -0.1184845119714737, 0.8406778573989868, -0.042464569211006165, 0.16741059720516205, -0.17550215125083923, -0.2877820134162903, 1.2425167560577393, 0.35506850481033325, -0.36411336064338684, -0.12799760699272156, 0.7471523880958557, 0.43694838881492615, 1.3210968971252441, 0.1530001014471054, 0.356362909078598, 0.7228000164031982, 1.1302392482757568, -1.0540056228637695, 0.6725332736968994, 0.7223231196403503, 0.9116823673248291, 1.2486116886138916, 1.4081398248672485, -1.9488496780395508, 0.13502930104732513, -1.0377017259597778, 0.6497159004211426, 0.36397403478622437, 0.768186092376709, -0.8739001750946045, 0.4388592541217804, 1.4183101654052734, 1.6716824769973755, 0.3970924913883209, 0.4395548105239868, 0.3418317139148712, 1.1514432430267334, -0.9422274827957153, 1.6904950141906738, 1.078934907913208, -0.827597439289093, -0.3570155203342438, -0.3042220175266266, -1.2100181579589844, 1.5867277383804321, 0.2779967486858368, 1.243541955947876, 0.185747429728508, 0.7413432598114014, 0.8870947360992432, 0.3436107933521271, 0.9564417004585266, 0.9329180717468262, 1.4139533042907715, -0.1472022831439972, -1.1534620523452759, -0.565631091594696, -0.636046826839447, 0.3497627377510071, -0.405350923538208, -1.097271203994751, -0.8470376133918762, -0.7113251686096191, -2.2823739051818848, 0.2303532063961029, 0.3857623040676117, -0.44775068759918213, -0.0029310015961527824, -0.5950545072555542, -0.7496829032897949, 1.2063484191894531, -0.08037443459033966, -0.978803277015686, 1.0455623865127563, 0.5857494473457336, 0.06598029285669327, -0.7995694875717163, -0.439448744058609, -0.6718198657035828, 0.4716806709766388, -1.3258395195007324, -1.7515250444412231, -0.2451251596212387, -0.5363542437553406, 0.7406294941902161, 0.36269867420196533, 0.8190075159072876, -1.0457074642181396, -2.012693166732788, 1.557625412940979, -0.761235773563385, 0.1529676616191864, -0.2149270623922348, -0.2912173271179199, -1.86319899559021, -0.8561685085296631, -2.0868468284606934, -1.0269384384155273, -1.777999758720398, -0.6012528538703918, -0.5389994978904724, -0.9306659698486328, -0.5847219228744507, -0.48282837867736816, 0.9736027121543884, -0.5521314144134521, -1.1280531883239746, 0.7652178406715393, -0.05535244196653366, 0.59026700258255, -0.463198721408844, -1.6525746583938599, -0.5304173827171326, -0.3181081712245941, -1.3391923904418945, -0.9468777179718018, 0.24759958684444427, 0.643539309501648, -1.675048589706421, 0.8764575123786926, -0.8746156096458435, -1.3518781661987305, -1.004231572151184, -0.13541917502880096, 0.164182648062706, 0.8355482220649719, -0.9157053232192993, -0.48509421944618225, -0.49885985255241394, 0.6719857454299927, -1.2525856494903564, -0.31435367465019226, -0.5255485773086548, 0.3222558796405792, -0.651729941368103, -0.7232498526573181, -0.20240753889083862, 0.6344277858734131, 0.08265058696269989, 0.11698711663484573, 0.6587986946105957, -0.893618106842041, 0.3275293707847595, -1.107828974723816, -1.9410697221755981, -0.7116909623146057, -0.6418048739433289, 0.04615357145667076, 0.24975737929344177, 0.6001772880554199, -1.1900804042816162, -1.3755544424057007, 1.720870018005371, -2.6076841354370117, -0.16888347268104553, 0.22956077754497528, -0.4007163643836975, -0.22873961925506592, 0.6020514369010925, 0.7085375189781189, -1.2689430713653564, 0.402508407831192, 0.1654372215270996, -1.001103401184082, -0.16432055830955505, 0.022325940430164337, 0.8425105214118958, -2.6392288208007812, 0.7437794208526611, 0.29839175939559937, 0.401294082403183, 0.08057788014411926, -0.2682655453681946, 1.5444432497024536, 0.5978733897209167, -0.393036425113678, -0.17401652038097382, -0.22473730146884918, -0.15969307720661163, -1.0540663003921509, -1.613635778427124, -0.38354426622390747, -0.2809670865535736, -0.9227701425552368, 0.7307320833206177, -1.3021466732025146, -0.04376747086644173, 1.4652700424194336, 0.47274526953697205, 0.5907618999481201, -1.289624571800232, 0.39404183626174927, -0.8536776304244995, 0.5076780915260315, 0.48863115906715393, -0.7578667998313904, -1.8010313510894775, -0.4648473560810089, 0.6634225845336914, 0.6754546761512756, -0.6384646892547607, 0.34931862354278564, -0.7632729411125183, -0.9520545601844788, 0.7546859979629517, -0.093141108751297, -0.18069487810134888, -0.82756507396698, -0.6983396410942078, 1.7163894176483154, 0.8303170204162598, -0.0404122993350029, 0.6868900656700134, -0.08526836335659027, 0.29495665431022644, -1.2965638637542725, 1.1320922374725342, -2.350628614425659, -0.10686766356229782, -0.2433195859193802, 0.47115665674209595, 0.005490703508257866, -0.6970155239105225, -0.040239058434963226, -0.31085342168807983, 0.8983352184295654, 1.0013368129730225, 0.6038374900817871, 0.5794508457183838, -0.5902421474456787, 0.3033856153488159, -1.7248270511627197, -1.3388675451278687, -1.2851741313934326, 0.02074624039232731, -1.243621587753296, -0.6601951122283936, -0.6580162048339844, -0.6636818051338196, -0.9231103658676147, -0.4576595425605774, -1.3162611722946167, -1.8327558040618896, -1.7475899457931519, -0.13451793789863586, 3.0271010398864746, 0.09473904967308044, 0.13218359649181366, 0.33525893092155457, 1.1228835582733154, -0.052757829427719116, -0.6808229684829712, -0.013350304216146469, 0.23892322182655334, -0.8412392735481262, 0.5844506621360779, -0.7795257568359375, 1.161475658416748, 0.30390626192092896, 0.2987213730812073, 1.015614628791809, -1.608994722366333, -1.1341309547424316, 0.1112254410982132, -1.498022437095642, -1.2383229732513428, 0.3781200647354126, 0.3363261818885803, 1.1053844690322876, 0.273431658744812, -0.43383926153182983, 1.5771673917770386, -0.19864733517169952, 0.5569968223571777, -2.496567964553833, 0.623748242855072, -0.10883013904094696, 0.36689886450767517, -0.30034029483795166, 1.8158533573150635, -0.8210768699645996, 1.610000729560852, 0.2016015499830246, -1.166249394416809, 0.7738175392150879, 0.41156062483787537, 1.0234781503677368, -2.3743577003479004, -1.7828400135040283, -0.3465026915073395, 0.26724767684936523, -0.3155858814716339, 1.9303579330444336, -0.06760476529598236, 0.4299369752407074, -0.2849401831626892, -0.46239736676216125, -0.36949416995048523, -0.7079477310180664, 0.3567909598350525, -0.12333253771066666, 1.3094614744186401, -1.145604133605957, 0.8092211484909058, -1.7290714979171753, 0.7900136113166809, 1.1501752138137817, -0.5777007341384888, 3.5613739490509033, 1.3061740398406982, 0.8964909911155701, -0.08722803741693497, -0.49614042043685913, 0.958248496055603, 0.8602233529090881, -1.965760588645935, 0.9235294461250305, -2.2271578311920166, 0.5747610330581665, 1.0227700471878052, -0.715227484703064, 0.5969293117523193, 0.30374765396118164, -1.4735748767852783, 0.1402273029088974, -1.0423609018325806, 0.8921065926551819, 0.4813394248485565, -0.8782944679260254, -0.14647041261196136, -0.5098234415054321, 0.6511448621749878, 0.12718544900417328, -1.1135903596878052, -0.5757835507392883, -0.051214784383773804, 0.19140975177288055, 0.2446800023317337, 0.8422605991363525, 0.05615559592843056, 0.46003153920173645, -0.3351714313030243, 0.5939040184020996, 1.470819354057312, 0.10646270960569382, -0.5546694993972778, 0.5237879753112793, 1.3856536149978638, -0.6214033365249634, 1.2960435152053833, 1.1737887859344482, -0.36828410625457764, 1.6561998128890991, 0.5436393618583679, -0.9334803223609924, -0.9582352042198181, 1.8607373237609863, -1.871572494506836, -0.688456118106842, 0.18087084591388702, -1.910484790802002, -0.44185298681259155, 0.9137914776802063, -1.977063536643982, -0.6079026460647583, -0.44087791442871094, 0.6490601301193237, 0.5645822286605835, -2.2004613876342773, 0.08761085569858551, -1.368585228919983, 0.1176183819770813, -0.9557337760925293, -0.2727753520011902, -0.04093597084283829, 1.3133422136306763, 0.5359783172607422, -0.2356923669576645, -0.46507319808006287, 0.09413060545921326, 0.10269629210233688, 0.010320909321308136, -1.4341952800750732, 2.2352325916290283, -0.18620344996452332, -0.01612764038145542, 0.5452673435211182, -0.41847723722457886, 1.0336617231369019, -0.01632688194513321, -0.3867238759994507, 0.36122408509254456, 1.057711124420166, -0.7506144642829895, -0.515093207359314, -0.046523936092853546, -0.5539655685424805, 0.43500521779060364, -1.3398406505584717, -0.6378353238105774, -1.5292158126831055, -0.22863776981830597, 0.8500752449035645, 0.052901748567819595, -2.143617630004883, 1.9187179803848267, -0.9650037288665771, 0.6016373634338379, 0.43779394030570984, 0.43486469984054565, -0.4049725830554962, 0.24148571491241455, 0.3497435748577118, 1.2253718376159668, -0.2970028519630432, 0.4304054081439972, 0.23745593428611755, 1.1859346628189087, 1.3732237815856934, 0.4817204773426056, -0.8843827247619629, -0.6378291845321655, -0.6405483484268188, -0.40849414467811584, -1.3753575086593628, -0.3852391541004181, -2.054105281829834, 0.846538245677948, -0.7489492297172546, -0.4298016428947449, 2.1172375679016113, 1.4194540977478027, 0.9563995599746704, -0.9966990947723389, 2.110475540161133, -0.5308181643486023, -0.35206031799316406, -0.5059014558792114, -0.7324122190475464, -0.2895359992980957, -1.3520255088806152, 0.3286030888557434, 0.2020939737558365, 0.25174447894096375, 0.4517744183540344, 0.3121146261692047, -0.038215186446905136, -0.47284018993377686, 0.3022922873497009, -0.3661707937717438, 0.16021762788295746, 1.3958903551101685, 0.7319891452789307, 1.9483994245529175, -0.43961989879608154, -0.4534291923046112, -1.1461400985717773, 0.1506310999393463, 0.3027222454547882, 0.8907101154327393, -0.479547917842865, -2.270932912826538, -0.07223884016275406, -0.8187171816825867, 1.0358269214630127, 0.02454676851630211, -1.8454291820526123, 2.2016842365264893, -0.7089025378227234, -2.066155195236206, -0.096694715321064, 0.109630286693573, 0.41779857873916626, -2.0658440589904785, 0.38916146755218506, 0.5579811930656433, -1.1571059226989746, 0.08460278064012527, -1.0053709745407104, 1.045555591583252, -1.5474841594696045, -0.7689399123191833, 0.6412375569343567, 1.111703872680664, 1.134325385093689, 0.5455910563468933, -0.17149591445922852, 2.154860734939575, -0.4401101768016815, -0.5563513040542603, -1.4619287252426147, -0.17700983583927155, 0.22769306600093842, 1.3483991622924805, 0.4130899906158447, 0.7126333713531494, 0.5713827610015869, -1.0824193954467773, 0.8329146504402161, -1.376570224761963, -0.2735065221786499, -0.13133321702480316, -1.127389907836914, -0.15668359398841858, 0.48814642429351807, -1.52784264087677, -0.6837963461875916, 0.117582768201828, -0.9867669343948364, -0.3115118145942688, 0.8244286179542542, -1.9079654216766357, 0.699665904045105, 0.11579729616641998, 1.290921688079834, -1.803197979927063, 1.774094581604004, -0.6852564811706543, 0.19071438908576965, -0.4251561462879181, 1.4902708530426025, 0.11449927091598511, -0.7154887318611145, -1.2537052631378174, -2.463785171508789, -0.7217411398887634, 0.9667536616325378, -0.5342167019844055, -1.259546160697937, 0.5206980109214783, 1.7841862440109253, 0.8403568863868713, 0.020385542884469032, -0.4281827509403229, 1.4551507234573364, 0.4265593886375427, 0.49767038226127625, 1.3393104076385498, 0.6592368483543396, 0.9497199058532715, -1.3328430652618408, 0.36315658688545227, -1.104721188545227, 0.8355710506439209, 0.3712257146835327, -1.487646222114563, 1.507547378540039, 0.821237325668335, -0.4616927206516266, 0.01194161269813776, 3.081784248352051, 0.8943060040473938, 0.6997042298316956, -1.736968994140625, 0.22557099163532257, -1.2846871614456177, 0.0015366615261882544, 1.2657808065414429, -1.0868823528289795, 0.42526504397392273, 0.07134935259819031, 0.8402854204177856, 0.7754358053207397, 1.0277332067489624, -0.7911194562911987, 1.967140793800354, -0.432524710893631, -0.29330962896347046, -2.016469955444336, -1.4695206880569458, 0.2798541486263275, 1.5991463661193848, 1.067226767539978, -0.020635876804590225, 0.6404042840003967, 0.13423380255699158, -1.5814915895462036, -1.8483319282531738, 0.8764765858650208, 0.21013136208057404, 0.13404205441474915, -0.3812474310398102, -1.5709433555603027, 0.43467476963996887, 0.9858474135398865, 0.2409607470035553, -0.29772159457206726, 1.0452312231063843, -0.797900915145874, 0.7174824476242065, 0.19644956290721893, -0.14967648684978485, -1.1986374855041504, 0.08842895179986954, -0.05035651847720146, 2.1487858295440674, -0.037564925849437714, 0.7735411524772644, 0.5512554049491882, -1.5982754230499268, -0.974509596824646, -1.3760937452316284, -0.49714332818984985, -1.0441759824752808, 0.5116959810256958, -1.2674534320831299, -1.1155186891555786, 1.0161691904067993, 2.987029790878296, -0.8732265830039978, -1.2856009006500244, 0.26723238825798035, 0.3597010374069214, 0.6250424385070801, 0.02452874928712845, -0.3168465793132782, -0.9478220343589783, 0.46563026309013367, -1.5809428691864014, 0.4708161950111389, -0.6600214242935181, 0.7395182847976685, -0.18150988221168518, 0.8854843378067017, -0.8371604084968567, 1.419791579246521, 0.6736201643943787, -0.5925959348678589, -0.5115509033203125, 1.4251543283462524, -2.2101213932037354, -0.04901507496833801, 0.13041432201862335, -0.141665980219841, -1.1714848279953003, -0.47929418087005615, 0.9454832673072815, -0.050887007266283035, 1.1742416620254517, 0.30552050471305847, 0.5030154585838318, 0.027494065463542938, 0.4288122057914734, -1.2080808877944946, -1.246180534362793, 1.058038353919983, -0.905489981174469, -0.2874615788459778, -1.3696175813674927, 0.4212721586227417, 0.3552899956703186, -1.5100940465927124, 0.22952726483345032, -2.6023190021514893, 1.216107964515686, 1.0911502838134766, -0.9301717281341553, 0.1262054443359375, -1.4298570156097412, 0.24116180837154388, 0.20316970348358154, 0.7402478456497192, 1.8016836643218994, 2.746689796447754, -0.07759859412908554, 0.6051437258720398, -1.5620893239974976, 0.5438789129257202, 1.1394387483596802, -0.32851099967956543, 0.5588134527206421, -1.5879260301589966, -0.504817008972168, -0.13920074701309204, -0.2987153232097626, 1.7580939531326294, 0.8731194138526917, -0.1123400554060936, 1.7728770971298218, 0.710101842880249, -1.2330286502838135, -0.04239778220653534, -0.3617388606071472, -0.09061839431524277, 1.5490922927856445, 1.5285221338272095, 0.23472575843334198, -1.1035553216934204, 1.4156512022018433, 0.03338855877518654, 1.6409577131271362, 0.18838077783584595, 0.5992078185081482, -0.7680897116661072, -0.8810076117515564, 1.219818115234375, 1.4888161420822144, 1.1924632787704468, -0.24793103337287903, -1.285313367843628, 0.008739122189581394, -0.3976222574710846, -1.5244700908660889, 0.5982264280319214, 0.2969367802143097, 1.8811137676239014, -0.4393315315246582, -0.07243361324071884, 0.4307537376880646, -1.2991130352020264, -0.8205446600914001, -1.136218786239624, -0.17734238505363464, 1.0246000289916992, -0.9524304270744324, 2.71386981010437, -0.9197431802749634, 0.6134259700775146, -1.4606499671936035, 1.3637908697128296, 1.188753604888916, -1.2211003303527832, -0.3157957196235657, 0.8659793138504028, 0.1927637755870819, 0.25332871079444885, -0.601906955242157, -0.08335482329130173, 0.5071074366569519, 1.0521798133850098, 0.6783718466758728, 0.5150684118270874, -0.8990784287452698, 0.2574327886104584, 0.666423499584198, -0.775894820690155, -0.45148608088493347, 0.6766011714935303, 0.18062113225460052, 0.1489560604095459, -0.532436728477478, -0.6057082414627075, -0.9336243867874146, 1.1627517938613892, -0.058624107390642166, -2.5524630546569824, 0.3654899001121521, 1.0957461595535278, -0.4127516746520996, -1.2690250873565674, 0.20085352659225464, -0.11172930151224136, 0.8281837701797485, 0.15198558568954468, -1.5013153553009033, 0.801598310470581, -0.8390493392944336, 1.2652312517166138, -0.6738548874855042, -0.14586268365383148, -0.04294488579034805, -1.2869963645935059, -0.24229031801223755, 0.28758567571640015, -0.03861204907298088, 0.2378402054309845, -0.5659772157669067, -1.360779881477356, 0.14310510456562042, -0.30601122975349426, 0.9995903372764587, -0.21469536423683167, -0.2636954188346863, -1.453881025314331, -1.2415767908096313, 1.0258338451385498, 1.8070167303085327, -0.7246366143226624, 0.3123863637447357, 0.34035688638687134, -1.837354063987732, 0.2151949405670166, 0.3273187577724457, -0.9698058366775513, 0.012248505838215351, -0.5144546627998352, 0.6095823049545288, 0.8387430906295776, 0.7769259214401245, -0.6212754249572754, -0.7807248830795288, -0.45203179121017456, 0.5148680210113525, -0.7615104913711548, -0.13849171996116638, -0.16729266941547394, -1.4806658029556274, 0.8344802260398865, 0.035682570189237595, 0.9021099805831909, -0.7926914691925049, 0.6583417654037476, 0.7387159466743469, 0.9360882639884949, -1.5135964155197144, 1.4181264638900757, -1.453631043434143, -0.24750745296478271, 0.12124378979206085, 0.48730722069740295, -0.4932628571987152, 0.586286187171936, -0.6183293461799622, -0.24022136628627777, 0.4774019122123718, 1.1169220209121704, 0.8603614568710327, -1.1023931503295898, 1.911910891532898, -0.7860018014907837, 0.0492364875972271, 1.0656931400299072, -0.8841094970703125, 0.2862023711204529, 1.1968153715133667, 0.3624415695667267, -2.4728260040283203, 0.2624813914299011, -0.3709917366504669, -1.0751019716262817, 0.6998291611671448, 0.8651341199874878, 0.5663582682609558, -0.7500767111778259, 0.5579073429107666, -0.34170985221862793, -0.43112245202064514, 0.9898431301116943, 1.375663161277771, 0.6041696071624756, 0.8974232077598572, -1.4423465728759766, -0.146004781126976, 0.36206501722335815, -0.6294392943382263, -0.7247300744056702, 1.8961844444274902, 0.36005276441574097, 1.758089303970337, 0.8116143345832825, 0.19606944918632507, 1.225017786026001, 0.24719807505607605, 0.5166503190994263, -0.5039921998977661, 1.263968586921692, -0.2701127231121063, -1.4052703380584717, 0.004644496366381645, -0.12068377435207367, -1.5668814182281494, -0.5205374360084534, 0.5470494627952576, -1.3374508619308472, -0.9076831340789795, 1.4130940437316895, -1.1531693935394287, 1.2801512479782104, -0.17440755665302277, 1.3300518989562988, 0.873180627822876, 0.6276702880859375, 1.233638048171997, -2.1562702655792236, 0.7468004822731018, 0.10604444891214371, 0.7674656510353088, -0.39587780833244324, 1.2166115045547485, -0.047546058893203735, 0.4586966335773468, -1.4227049350738525, 0.2275945097208023, 2.1374526023864746, 0.5809855461120605, -0.7248849868774414, 1.4631849527359009, -1.033401608467102, -2.1635546684265137, -0.732636570930481, 1.2213937044143677, -0.9959121346473694, -0.11450919508934021, -1.2909437417984009, -0.7720411419868469, -0.25035765767097473, -0.990262508392334, -1.6378445625305176, -1.4287601709365845, 0.11341075599193573, 0.8799368143081665, 1.3285149335861206, 1.1876765489578247, 0.10635031014680862, 0.6679169535636902, 0.06618563085794449, 0.2735646069049835, 0.4513254165649414, 0.41901397705078125, -1.1153182983398438, 0.14133363962173462, -0.7183240056037903, 0.6352181434631348, -0.9499621391296387, -0.24978426098823547, -0.4919669032096863, -0.4503287076950073, 0.7373222708702087, 0.5888909697532654, 2.4350674152374268, -1.1886197328567505, 0.6793760657310486, 0.3563167154788971, 2.592095375061035, 0.06870831549167633, -0.05245095118880272, 1.3707562685012817, -0.895982563495636, -0.2991117238998413, 0.03818468749523163, 0.2605166435241699, -0.964653730392456, 0.43097731471061707, 0.4808904230594635, 0.11266990751028061, -0.9240456819534302, 0.671570360660553, 0.26286253333091736, -1.1406090259552002, -1.8899240493774414, -0.6836390495300293, -0.07428302615880966, -0.7749966979026794, -0.46295347809791565, 0.40591078996658325, -0.5771378874778748, 0.9606651663780212, -0.845920205116272, -2.0763325691223145, 0.874619722366333, 0.08658565580844879, -1.1761654615402222, 1.819083333015442, -1.4460346698760986, 0.06562507152557373, 0.4538077116012573, 0.14154911041259766, 0.9018211364746094, 0.25354281067848206, -0.511141300201416, 1.228920340538025, -1.1345560550689697, -0.05925430357456207, -0.5550615787506104, -1.219153881072998, 0.007644210942089558, -0.9645874500274658, -1.4124584197998047, 1.3264663219451904, -0.5944786667823792, -1.6065037250518799, -0.40353459119796753, -0.8799612522125244, -0.06454595923423767, 0.26417094469070435, 1.5733214616775513, -1.1394941806793213, 0.6806536912918091, -0.7054954767227173, -0.8440096378326416, 1.9610949754714966, 0.5988538265228271, 1.9001131057739258, -0.5411152243614197, 0.2030496448278427, -0.015655329450964928, -0.12481115758419037, 1.9900522232055664, 0.19106966257095337, -1.2240546941757202, 0.17172540724277496, 0.36455851793289185, -0.6671581268310547, 0.7396032214164734, -0.3227662444114685, -1.898641586303711, -0.31366944313049316, -1.1882004737854004, -0.569997251033783, 0.5116829872131348, 1.482162356376648, 0.8563802242279053, 0.6906698346138, 0.9406430125236511, 1.226456642150879, 0.30606430768966675, 1.6124818325042725, 1.2600185871124268, -0.8404487371444702, -0.7530298829078674, 1.1965376138687134, 0.13519544899463654, 0.4657243490219116, -0.24577364325523376, -0.192820742726326, -0.17542503774166107, 1.5565645694732666, 1.8885376453399658, 0.6555439233779907, -0.23280270397663116, 0.5577966570854187, 0.7248297929763794, -1.293821096420288, -0.7136482000350952, -0.24480664730072021, 0.38164377212524414, -0.29623115062713623, 0.044855114072561264, -1.0351251363754272, 0.19244468212127686, -0.14216981828212738, -1.5339627265930176, -0.31281834840774536, 0.7359951734542847, 0.4725227952003479, 1.8632105588912964, 1.5405604839324951, -1.0420666933059692, -0.31417712569236755, -1.8707088232040405, -1.2394481897354126, -0.2421218752861023, -0.09015628695487976, -1.082897424697876, 2.2109851837158203, 0.2519201636314392, -0.8824571967124939, 1.285772681236267, -0.8200033903121948, -0.14950482547283173, 0.7121877074241638, 0.8972626328468323, -0.375990092754364, -0.4360258877277374, 0.35212305188179016, 0.6770857572555542, -1.451518177986145, 1.1308618783950806, -0.6528480052947998, -0.45661383867263794, -1.0834970474243164, 0.5176032781600952, -0.5216027498245239, -0.8954669237136841, 0.5867773294448853, 1.7455902099609375, 1.774274468421936, 1.1161073446273804, -0.31214091181755066, -2.3267061710357666, 0.36182093620300293, 0.474651038646698, -0.7664347290992737, 0.41148024797439575, 0.48009175062179565, -0.06188786402344704, 1.0133916139602661, 0.4883454144001007, -0.6077641844749451, -0.0246119424700737, -1.2592308521270752, 0.5213471055030823, -0.07819905877113342, -0.015180771239101887, -0.9762517809867859, 0.5566759705543518, 0.1327713131904602, 0.4479120373725891, 0.39575067162513733, 2.346251964569092, -1.6602500677108765, -0.7121894359588623, -1.7445430755615234, 0.843716025352478, 0.37258654832839966, 0.3024618625640869, 0.33519190549850464, -1.817513346672058, 0.13624875247478485, 1.0628507137298584, 1.2244888544082642, 0.4620773196220398, -0.10839042067527771, 0.1986604630947113, 0.8560271263122559, 0.8884536027908325, -0.42774003744125366, -0.7831143736839294, -1.1448527574539185, -1.2697935104370117, 1.258880853652954, 0.08170025795698166, 0.6196658611297607, -1.3904168605804443, -0.9694796204566956, 0.02203252539038658, 0.08139389753341675, 0.5438872575759888, -1.1324479579925537, 0.48584529757499695, 0.17224621772766113, 1.2650200128555298, -1.8061141967773438, -0.7765689492225647, 0.03483354672789574, 1.5639528036117554, 0.7888898253440857, -0.3972817063331604, -1.317698359489441, -1.0360041856765747, -0.015722043812274933, 0.8480256795883179, -1.7586281299591064, 0.8153440356254578, -1.5078998804092407, -1.5702449083328247, 1.0555287599563599, 0.24666182696819305, -1.249794602394104, -0.4472799301147461, 1.7925543785095215, -1.2427948713302612, -1.6125049591064453, -0.7463582158088684, -0.40119650959968567, 0.46888354420661926, 0.3287055790424347, 1.3163845539093018, 0.6772045493125916, -2.740649700164795, -0.2553398013114929, 0.27851414680480957, -0.9349656105041504, -0.22184988856315613, -0.5117266774177551, -0.6179840564727783, -0.1254546046257019, -0.3819997310638428, 2.0034706592559814, -0.7686436772346497, -0.7949676513671875, 0.3376835882663727, -1.6687698364257812, -0.036547884345054626, 1.3057631254196167, -0.4163353145122528, 0.11367689818143845, -0.2491605281829834, 0.7149948477745056, -0.8368237018585205, 0.19157780706882477, -0.964799165725708, -0.3149721920490265, -1.4501607418060303, 0.8854621052742004, -1.4575929641723633, 0.08667907118797302, 1.4975146055221558, 0.9028213620185852, -0.5858430862426758, -0.09648363292217255, -0.7535565495491028, -0.5010713934898376, 0.45245808362960815, 0.012294015847146511, 0.35657966136932373, 0.7948764562606812, 1.3151795864105225, -1.1513036489486694, -1.551964282989502, -0.3424838185310364, 1.2153208255767822, -0.6725789308547974, -1.0188204050064087, 0.5930126905441284, 1.1948912143707275, -1.6755483150482178, -1.4465389251708984, 1.09397292137146, 1.5423619747161865, -3.6990420818328857, 0.4819594919681549, 0.5450445413589478, 0.4721829891204834, -0.5309075117111206, 0.12672393023967743, 0.5961922407150269, -0.9911426901817322, 1.3185704946517944, -1.0568474531173706, -1.0297003984451294, -1.1769471168518066, 0.0037812425289303064, -0.5051408410072327, -0.21619942784309387, 0.33091261982917786, 1.521904468536377, -0.21798966825008392, -0.6206924915313721, -0.4115421175956726, 0.11101074516773224, -1.0621942281723022, 0.20027022063732147, -2.3128507137298584, 1.1464852094650269, -1.9513647556304932, -0.12019049376249313, 0.35173681378364563, 0.15005286037921906, 0.1969713419675827, 1.3842207193374634, 0.6253087520599365, 1.3237358331680298, 0.18713180720806122, -0.1703038364648819, -0.5638731718063354, -0.24531204998493195, 1.4155796766281128, 0.18164344131946564, 0.9603356122970581, -1.1334617137908936, 0.19313523173332214, 0.5149215459823608, 0.06723863631486893, -0.29171133041381836, 2.166257619857788, 0.7378217577934265, -1.6032382249832153, 0.4538962244987488, 0.2925877869129181, 0.9234319925308228, -2.1212151050567627, 1.7462496757507324, -1.7674278020858765, 0.9312614798545837, -0.04347606003284454, 0.5755004286766052, 0.7547353506088257, 0.8276327848434448, 1.3980833292007446, -0.49861758947372437, -0.27953410148620605, -0.27662524580955505, 0.7741713523864746, 0.30142301321029663, 0.04509309306740761, -0.3105379343032837, -0.3461964726448059, -0.33001843094825745, -1.8421217203140259, -1.715935468673706, 1.4304821491241455, -1.5277154445648193, 1.4097933769226074, -0.5754877328872681, -0.16561324894428253, 2.069135904312134, 0.015995489433407784, -1.5208990573883057, 0.4875734746456146, -0.39537206292152405, 0.8731324672698975, 0.6076760292053223, 0.37247729301452637, -1.2777187824249268, 0.5275216698646545, -0.4593923091888428, 0.7586280703544617, -0.2718021869659424, 0.48845067620277405, 1.1447395086288452, -0.9862456917762756, 2.087709665298462, -0.569678544998169, 0.6338865756988525, -2.9397997856140137, -0.46591976284980774, -1.451462984085083, 0.8412827849388123, 0.02436225116252899, 0.07049494236707687, 1.2217020988464355, 0.3964082896709442, 0.7777866125106812, 0.48987263441085815, 0.8106490969657898, 1.1303569078445435, 0.5878821015357971, 0.13192026317119598, 1.4871761798858643, -1.520721197128296, 0.8151045441627502, 0.2194141447544098, 0.1671905815601349, 1.66170072555542, -0.29880163073539734, 0.11822451651096344, -0.4585299789905548, -1.2899349927902222, 1.160736083984375, 1.8810187578201294, -0.708901047706604, -1.7130144834518433, 1.4009368419647217, 0.8565692901611328, -0.5276041030883789, 1.0281040668487549, -1.0054999589920044, -0.43352341651916504, -1.4658358097076416, 0.6804315447807312, -1.3105735778808594, 1.1210765838623047, 0.7845911979675293, -1.4007858037948608, -2.165930986404419, -1.711472511291504, 1.440849781036377, -0.04935452714562416, 3.0098929405212402, 1.96397864818573, -0.5333273410797119, -1.2508865594863892, -0.8641866445541382, 0.3164512813091278, 1.3088550567626953, -0.9382851719856262, -0.8693405389785767, 1.5278539657592773, 1.0658068656921387, -1.6647125482559204, -0.5873497724533081, 1.1909772157669067, 0.4152063727378845, 1.0673387050628662, -0.17754937708377838, -0.1439955234527588, 0.824470579624176, -0.20710885524749756, -0.038572460412979126, 0.27911943197250366, 1.9070972204208374, -0.39433613419532776, 0.7251296043395996, -0.09780700504779816, -0.6094712615013123, 0.7099243402481079, -0.856526255607605, 1.5567537546157837, -0.11990942806005478, 0.6658657789230347, 2.036801815032959, 0.342941015958786, -1.1794228553771973, 0.9940159320831299, 1.7625811100006104, -0.0678764134645462, -0.3940020203590393, -1.1299551725387573, -0.4605576992034912, -1.386254072189331, 1.0377907752990723, -0.5008681416511536, -1.0791525840759277, -0.012997605837881565, 0.5407549738883972, 0.14176678657531738, -0.6851532459259033, -0.4269087612628937, -0.5959010124206543, -1.4231687784194946, -1.0790355205535889, -0.9049251079559326, 0.6466757655143738, -0.9833264946937561, 0.43764936923980713, 0.3155507743358612, -1.7870135307312012, 1.2866202592849731, 0.27463653683662415, -1.1470423936843872, -1.291176438331604, 0.9457209706306458, 0.8459200263023376, 0.9743867516517639, -2.5712947845458984, 0.8295427560806274, 1.4274775981903076, -1.1995558738708496, -0.23597903549671173, -1.2023513317108154, 0.8201639652252197, -0.9935081005096436, 0.8836703896522522, 1.0077701807022095, 1.0268616676330566, 0.18511056900024414, -1.23578679561615, -1.1148267984390259, 1.0803112983703613, 0.7565343379974365, -0.5385770201683044, 1.7482140064239502, -1.9557870626449585, -0.42810946702957153, -0.0008007276919670403, 0.2614201307296753, 0.4804191589355469, 1.3780909776687622, 0.8433747291564941, -2.678032875061035, 0.6677515506744385, -1.0760128498077393, 0.9666473269462585, -0.8606531023979187, 1.0747672319412231, 0.5751677751541138, 0.15079393982887268, 0.16737833619117737, 0.9082800149917603, 0.6942921280860901, -1.4818214178085327, 0.9165138602256775, -0.6152479648590088, -0.9899073243141174, -0.6295152306556702, 2.0460195541381836, -0.1580977886915207, 0.19453419744968414, 0.046078577637672424, -1.7586461305618286, 1.3159898519515991, -0.4975410997867584, -2.3290438652038574, -0.3051280975341797, 0.9915889501571655, 1.6160694360733032, -0.39519524574279785, 1.2670148611068726, -0.4530748128890991, -0.15898475050926208, 0.8093795776367188, 0.05771531164646149, -0.9843148589134216, 0.10370750725269318, 0.4545605480670929, 0.8056816458702087, -1.8032498359680176, -0.8537614941596985, -0.5043407678604126, 0.16235408186912537, -0.8457948565483093, 1.0580053329467773, -0.405353307723999, -0.6650158166885376, -0.6689910888671875, -0.5164340734481812, 0.7714230418205261, -1.301045298576355, 0.43956997990608215, 0.6741496920585632, -1.1941874027252197, 0.8150593638420105, 0.34176480770111084, -1.1355741024017334, 1.2146320343017578, -0.6258150339126587, -1.600571870803833, 0.8890888094902039, -0.05961062014102936, 1.8286397457122803, 1.2919378280639648, -0.5065739750862122, -0.007173486985266209, 0.2646131217479706, -2.4300835132598877, -1.1484915018081665, -0.6831836104393005, -1.2601792812347412, 1.2153480052947998, -0.40284210443496704, 0.22447125613689423, -0.8479255437850952, 0.005076170898973942, 0.050138041377067566, -0.2349199652671814, 2.2267520427703857, 1.6941461563110352, -1.1078392267227173, -2.245270252227783, -0.8985239863395691, 0.4877685606479645, -0.8559707999229431, -0.7395144104957581, -1.4215983152389526, 1.1350430250167847, 1.3680397272109985, 3.153759717941284, 0.7549452185630798, -0.7953699231147766, -0.8263064622879028, 0.40246427059173584, 0.47370028495788574, 0.04786150902509689, 0.15974439680576324, 0.6574006676673889, 0.3026118576526642, 1.0002422332763672, -1.183313012123108, 0.6458344459533691, 0.30507969856262207, -0.4895702004432678, -1.0598119497299194, -0.1008143424987793, 0.8230153918266296, 1.5965962409973145, -0.7761374115943909, 1.8702137470245361, 0.6536742448806763, 2.5738885402679443, 1.427031397819519, 0.9788144826889038, 0.6242195963859558, -1.1758313179016113, 0.3360106647014618, 1.6794244050979614, -1.6176962852478027, 0.07984323799610138, -0.8964236378669739, -0.14582259953022003, -2.282099723815918, -2.1357460021972656, -0.6368613839149475, -0.48495328426361084, 0.15514202415943146, -0.2741950452327728, -0.568466067314148, -0.635417103767395, -1.0836554765701294, 0.6021008491516113, -1.3394570350646973, -0.21907095611095428, 1.8506722450256348, 0.013115709647536278, -1.7970274686813354, -0.9916908144950867, 3.005366802215576, -0.6079307794570923, 1.538978934288025, -1.1389822959899902, 0.2370266616344452, -0.6834495067596436, -3.0026051998138428, -0.9922608733177185, 0.1526193916797638, -0.25651657581329346, -0.5311351418495178, 0.35924336314201355, 0.9597519040107727, -2.4322903156280518, -1.522969365119934, -1.6769732236862183, 0.09092656522989273, 0.5373790264129639, 1.152912974357605, -1.0974242687225342, -0.9949998259544373, -1.0278218984603882, 0.45403537154197693, -0.024670004844665527, 0.4300585687160492, -0.34595808386802673, -0.15591920912265778, 1.748347282409668, -0.12252064794301987, -1.459871768951416, -0.35001012682914734, -1.7386735677719116, -0.2140006572008133, 1.7994178533554077, 2.5473361015319824, 1.579811692237854, 1.030395269393921, 0.8634189963340759, -0.6126827597618103, -1.3516875505447388, -0.2032785266637802, 0.4965251684188843, -0.8030059337615967, 0.2135179191827774, -0.16760066151618958, 1.0259236097335815, -0.1983584463596344, -0.3327712416648865, 1.177500605583191, -0.5472663044929504, -1.3872630596160889, 0.11626709252595901, -1.0956000089645386, 1.041808009147644, -0.3676063120365143, -0.8816413879394531, -0.5180834531784058, -0.5853655338287354, 0.17606058716773987, 1.164458155632019, -1.336718201637268, -0.6613689064979553, 0.40212786197662354, 0.16583620011806488, 1.202641487121582, -0.5686755776405334, -1.1514182090759277, -1.4302797317504883, 1.4569098949432373, -0.7643589377403259, -2.139655113220215, -0.5774332880973816, -0.31268906593322754, 1.4338419437408447, -0.8305178284645081, 0.06067494675517082, 0.5251401662826538, -0.3022609353065491, 0.08570907264947891, 1.151827335357666, -2.2082176208496094, -1.032190203666687, 0.24651946127414703, -0.17049285769462585, 1.326603651046753, 0.682924747467041, -0.3773898184299469, -0.7583708167076111, 0.818274736404419, 1.213265061378479, -1.3427584171295166, 0.382218599319458, 0.9259458780288696, 0.18478122353553772, 1.062415599822998, 0.7517086863517761, 1.228936791419983, 1.2866835594177246, -0.5608770847320557, -1.0513207912445068, -0.4229944050312042, -2.056534767150879, -0.10434667021036148, 0.0432632677257061, -0.9139348268508911, -2.1075665950775146, 0.05461496859788895, 0.49213334918022156, 0.4648006856441498, -1.6909775733947754, 1.1045364141464233, -0.28506532311439514, 0.0884377658367157, -0.9646721482276917, -0.45262983441352844, -0.28243201971054077, -1.1172415018081665, -1.304364800453186, 1.2913858890533447, -1.0492926836013794, -0.4238247573375702, -0.03478844463825226, 1.9217352867126465, -1.9710084199905396, -1.174280047416687, 1.9140969514846802, 1.5361385345458984, 1.8865503072738647, -1.84092116355896, 0.11975281685590744, -0.6984719038009644, -0.05727490037679672, -0.7306919097900391, -0.43247804045677185, -1.6662886142730713, 1.2656525373458862, 1.3249906301498413, -0.5135566592216492, -0.5569681525230408, -0.2197074592113495, 0.042464956641197205, 0.281279057264328, -0.4676807224750519, 0.027392346411943436, -1.3382692337036133, -0.6717894077301025, -0.9595265984535217, 0.8292511701583862, 0.7302306294441223, 0.06091504171490669, -1.082169532775879, -0.18212604522705078, 0.7082582116127014, 1.6187363862991333, 0.658382773399353, 0.7950959205627441, 1.6639282703399658, -0.41596266627311707, 0.9776178598403931, 0.17509277164936066, -0.32665035128593445, -0.5926182270050049, 0.8095889091491699, -0.5229476690292358, 2.4619383811950684, -0.3292403221130371, 1.3137730360031128, -1.74606454372406, -1.0108245611190796, 0.7027367949485779, 0.04836318641901016, 0.29360660910606384, -0.9167594909667969, -0.6379808783531189, 0.2906687259674072, 0.9681942462921143, -0.8479950428009033, 0.7720344066619873, -0.4335669279098511, 0.755719006061554, -0.46462464332580566, -0.6769580841064453, -2.923339605331421, -0.27412736415863037, 0.8741111159324646, -0.17986980080604553, 0.35122251510620117, -0.5795970559120178, 0.6718090772628784, 0.0972880870103836, 0.415006160736084, -0.6356048583984375, 0.023063180968165398, 0.8605497479438782, -0.35720136761665344, 0.6842341423034668, -1.3242762088775635, -0.4852636158466339, 0.9742957949638367, -0.7269531488418579, -0.014042878523468971, -1.0005180835723877, -0.2991296052932739, 0.11406674981117249, -0.33530572056770325, -1.0768202543258667, 1.3319979906082153, -0.6411463022232056, -2.3603785037994385, -0.8141049146652222, 0.200271338224411, 0.3403467833995819, 0.3856259286403656, 0.9839684963226318, -0.7910593748092651, -0.6090916991233826, -0.5839974284172058, -0.35180771350860596, 0.8275722861289978, -0.34785106778144836, -0.8762142658233643, 0.5641381740570068, 0.7060225605964661, -1.2144893407821655, 1.0113978385925293, -0.2148267775774002, 0.5405350923538208, -1.3235610723495483, 0.24624767899513245, -0.16690964996814728, 1.1940422058105469, -1.6098130941390991, -0.6504095196723938, -0.2932431399822235, -1.9451732635498047, -0.6887162923812866, -0.2305462807416916, -2.4692270755767822, -0.5664952397346497, 0.1338917464017868, 0.3399493098258972, 0.6836364269256592, 0.46251246333122253, -1.279744029045105, -0.9213828444480896, 1.013674259185791, -0.008081506006419659, 1.4036579132080078, 0.1335466206073761, 1.1206556558609009, -0.7663206458091736, 0.481035441160202, -0.8191959261894226, 0.34309542179107666, 0.8825365304946899, -0.5272786021232605, 0.14848999679088593, 0.9091359376907349, -0.5949144959449768, -0.5983490943908691, -0.5682294964790344, -1.1377290487289429, -0.49780914187431335, -0.1143530085682869, 1.0149943828582764, 0.26202234625816345, -0.5262070298194885, 0.7847333550453186, 0.008750866167247295, 0.6063926815986633, 2.1667251586914062, -0.4595973491668701, -0.09574491530656815, 0.3280380964279175, 1.2106547355651855, -2.050201654434204, 1.354548454284668, -0.22203421592712402, -0.4449491500854492, -1.2273496389389038, 0.8168683648109436, 0.7635215520858765, 1.231399655342102, -1.6628540754318237, -1.9440956115722656, -0.1537116914987564, -0.3919219970703125, -0.5991958379745483, -0.34954142570495605, 0.22464069724082947, -0.40911558270454407, -2.6235265731811523, -0.5769472718238831, 1.2993403673171997, -0.21041710674762726, -0.5515425801277161, 1.0019376277923584, 1.3982161283493042, -2.053492784500122, -0.09178704023361206, -0.42722129821777344, 0.664370596408844, 0.29736125469207764, 1.8037261962890625, -0.6873728036880493, 0.785962700843811, -0.09202220290899277, -1.8327949047088623, 0.41309455037117004, -1.2409149408340454, 0.4520058333873749, -0.5217958688735962, 0.49791377782821655, -1.8679239749908447, -0.02237730473279953, -0.24007798731327057, 0.3242233991622925, 2.8576629161834717, 0.9611460566520691, -0.012486507184803486, 0.7958724498748779, 1.575904369354248, 1.583626389503479, 1.4512368440628052, 2.289722442626953, -0.26742392778396606, 0.4736485779285431, -2.0759661197662354, 1.435890555381775, -0.6366692781448364, 0.6208668947219849, -0.07339365780353546, 0.42208755016326904, 1.9886565208435059, -0.6818928122520447, -0.23249123990535736, -1.362876296043396, 1.3673210144042969, -1.7304309606552124, -1.4565023183822632, -0.25321444869041443, 0.6165845394134521, -0.6914817094802856, 0.6952551603317261, 0.8295767903327942, 0.1904841810464859, -1.226002812385559, 0.48831748962402344, -0.5669021010398865, -3.213043451309204, -0.16408957540988922, -0.8257747292518616, -0.4900934100151062, -0.1955776810646057, -0.13009801506996155, -0.04419715702533722, 0.4267033338546753, -0.15252211689949036, -0.9450140595436096, 0.2845921814441681, 0.871438205242157, 1.282550573348999, -0.4618777334690094, -1.3655328750610352, -1.0982824563980103, -0.7818112969398499, 0.26024892926216125, 0.16382445394992828, 1.3814799785614014, 0.08403719961643219, 0.6563994288444519, -0.07075007259845734, -0.5589127540588379, 0.0017130946507677436, -0.08333922922611237, -0.3201524615287781, 0.26707810163497925, -0.20675398409366608, -0.5498117804527283, 0.11062860488891602, -0.8472493290901184, -1.2862765789031982, -0.08782724291086197, 1.3809565305709839, 0.8893683552742004, -0.0850529670715332, -0.17562194168567657, 0.31356191635131836, -0.6968732476234436, 1.320364236831665, 1.3078205585479736, 0.18385303020477295, 1.5216399431228638, 1.07213294506073, -0.3808208107948303, 0.7020481824874878, -0.224257230758667, -2.1899471282958984, -0.8597103953361511, -0.8974117040634155, -1.726266622543335, -0.978847324848175, -0.3878189027309418, 1.3473620414733887, 1.006014347076416, -0.11633910983800888, -0.08383996039628983, -0.19182409346103668, 0.7652649879455566, 1.1531319618225098, -1.0170183181762695, -0.06414318084716797, -0.9756874442100525, 0.34444186091423035, 1.080227017402649, -0.9893010258674622, 0.007762709632515907, 0.15789766609668732, 0.661699652671814, 0.3217034935951233, -0.4667351245880127, -0.5898663997650146, 0.7695590257644653, -0.7693347334861755, -0.24530768394470215, 0.6624746918678284, 1.2478561401367188, 1.4366329908370972, 0.8940734267234802, -0.5743796825408936, 0.23741021752357483, -0.47264355421066284, 0.28330761194229126, -2.6601083278656006, 0.6107940077781677, 0.6583137512207031, 2.9248459339141846, 0.3252442479133606, -0.21992932260036469, 0.25109612941741943, 0.9850058555603027, 0.60549396276474, 0.2954319417476654, -0.3490193784236908, -1.3326555490493774, -0.7273761034011841, -1.4940184354782104, 0.5205016732215881, -0.5270110964775085, 0.9956151247024536, 2.1841988563537598, 0.3851359486579895, -1.2250374555587769, -1.0814841985702515, 0.5836172699928284, 0.7631968259811401, -0.3823864459991455, 0.3344651758670807, 0.5764251947402954, 0.46189987659454346, -0.9331075549125671, 0.3745521306991577, 0.19855834543704987, -0.8400079607963562, -0.10140977054834366, 0.7375427484512329, 0.15073257684707642, -1.4552605152130127, 1.451263189315796, 0.8357791900634766, 1.0490461587905884, 1.4152289628982544, -0.7585562467575073, 0.29137566685676575, -1.6929436922073364, 1.1852643489837646, -0.7994329333305359, -1.1862975358963013, 0.8063615560531616, -1.5407413244247437, -0.4138800799846649, -2.2790000438690186, 1.8491419553756714, 2.5229387283325195, 1.0260779857635498, -0.5078701972961426, 0.6933580636978149, 0.48830851912498474, -0.10421445220708847, -0.9329935908317566, -1.1313689947128296, 0.8613265752792358, 0.0909704640507698, -0.5167424082756042, -0.8686397671699524, 1.893654227256775, -0.5214899778366089, 1.1669397354125977, -1.733689785003662, 1.0975862741470337, 0.07620256394147873, 0.4727121591567993, -0.8911106586456299, -0.424912691116333, -0.2004937082529068, -1.2694569826126099, -1.1138330698013306, -1.0753506422042847, 0.6309054493904114, -0.10144180804491043, -0.9424936175346375, -0.9327714443206787, 0.4808815121650696, 0.35325923562049866, 0.41836807131767273, -0.08659224957227707, -0.9456019401550293, -1.096785068511963, -0.9686146974563599, 0.1234624981880188, 1.1025422811508179, 0.24570520222187042, 1.0538699626922607, 1.0701630115509033, 0.7948281168937683, -1.291552186012268, -0.737108051776886, -0.6799885034561157, 1.5594943761825562, -0.7746987342834473, 0.5039177536964417, 0.8283807039260864, -0.009160705842077732, -0.7139724493026733, 0.7299562692642212, -2.118088960647583, -0.9042849540710449, -1.13009512424469, -1.2390437126159668, -0.29199478030204773, 0.3184596300125122, -0.8448649644851685, 0.3180556893348694, -0.4226265847682953, -1.4800989627838135, 1.9874809980392456, 0.9329040050506592, -0.8128795623779297, -0.737667441368103, 0.17909684777259827, -0.270070344209671, -0.39204224944114685, 0.8059601783752441, 1.3501275777816772, -1.4235082864761353, -1.2457152605056763, -0.698023796081543, 0.8715112209320068, -1.2639590501785278, 1.139299988746643, 0.8883201479911804, 0.4699154496192932, 1.141789436340332, -1.5660983324050903, 0.5254888534545898, -0.4794059991836548, 0.88517826795578, -0.5403012633323669, -1.4669766426086426, -0.7928329110145569, 1.6036700010299683, 0.22504264116287231, 0.21807536482810974, 1.034673810005188, -0.27112865447998047, -1.546744465827942, -2.267291307449341, -0.18342836201190948, 0.35132575035095215, -0.6001712679862976, -0.37770453095436096, -0.3890498876571655, 0.674189567565918, -0.6127959489822388, 0.7521865367889404, 0.096961110830307, -0.21176549792289734, 1.3995434045791626, -0.019645294174551964, 0.1643563210964203, -0.2641673982143402, 0.3322364091873169, -1.0856560468673706, -0.7216845154762268, 0.7853544354438782, -0.44534146785736084, 0.4561787545681, 0.3308973014354706, -0.09047617763280869, -0.26610490679740906, 1.13493812084198, 1.9525129795074463, -0.49664434790611267, 0.48347094655036926, -1.2884316444396973, 0.061127517372369766, 0.7285657525062561, -0.8668779730796814, -0.30594104528427124, -0.40287861227989197, -1.3073182106018066, -1.1538926362991333, 0.2116934061050415, -0.2742445170879364, -0.9878891110420227, -1.201345443725586, -1.4324839115142822, -0.41008397936820984, -0.016601402312517166, 0.3147962689399719, 0.9183701276779175, 0.06496145576238632, -1.5371941328048706, 0.025670776143670082, -0.5064172148704529, -1.2505265474319458, 0.7268591523170471, 0.6217544674873352, 0.329845130443573, -0.6706042289733887, -0.36429962515830994, 0.2764151692390442, 0.6964346170425415, 0.0959421843290329, -0.37014973163604736, 0.09199471026659012, 0.7381665706634521, -0.440753310918808, -0.961662232875824, 1.683472990989685, -1.6883167028427124, -1.8044668436050415, 1.297235369682312, 0.3304690420627594, 0.7807584404945374, 0.5534898042678833, 1.6164995431900024, -1.0096796751022339, -1.0155457258224487, -0.520663857460022, 1.5801067352294922, 0.9028440713882446, 0.7808574438095093, -0.16870668530464172, -1.3162349462509155, -2.0840516090393066, 0.8370333313941956, 0.3556906282901764, -0.16315142810344696, 0.663378119468689, 0.7556813359260559, 0.5422397255897522, -2.0770933628082275, 1.1161655187606812, -0.5409088134765625, -0.14120803773403168, 0.6929000020027161, -0.6499976515769958, 1.3427892923355103, -0.23836787045001984, 0.20046323537826538, 0.7423864006996155, 0.7592893242835999, -0.13810333609580994, 0.21306996047496796, -0.8224732875823975, -1.5291939973831177, -1.1037368774414062, -0.5291832089424133, -0.09661480039358139, 0.2477758526802063, -2.5500292778015137, -1.4200749397277832, -0.5753350853919983, 0.41115516424179077, 0.11277634650468826, 1.9269405603408813, 0.8957124948501587, -0.9245930910110474, -0.019650960341095924, -0.8699221014976501, 0.861283540725708, -1.4570335149765015, 0.5411601066589355, -0.6631753444671631, -0.19176863133907318, 0.06952414661645889, -0.556646466255188, -0.48087263107299805, -0.38629278540611267, -0.18159589171409607, 0.6232479810714722, 0.7543498277664185, 0.18962031602859497, 0.747083842754364, 1.467429757118225, -0.8418270349502563, 0.2965952455997467, 0.40577375888824463, -1.8228217363357544, 0.1274651288986206, -0.773623526096344, 0.7938412427902222, -0.6520751118659973, -0.07956336438655853, -0.6166653633117676, -0.3862449824810028, -1.32981538772583, 1.0809046030044556, -1.0363954305648804, -0.4381411671638489, -0.15041819214820862, 0.3935662806034088, 0.6046974658966064, -0.5668630599975586, 0.19894783198833466, 1.7272344827651978, 0.41774868965148926, -0.937505841255188, -0.49619489908218384, 1.5241073369979858, 1.1814881563186646, 0.8023000359535217, 2.5961506366729736, 0.18185071647167206, -0.4087394177913666, -1.3024632930755615, -1.2754062414169312, -1.799890398979187, 2.1244516372680664, 0.4112926125526428, 1.7297563552856445, -2.2089102268218994, 0.8876376748085022, -0.5564510822296143, 0.9745684266090393, -0.6477425694465637, 0.22333167493343353, 0.7303387522697449, 0.2987820506095886, 1.370208978652954, 0.5897871851921082, -1.0479762554168701, -0.7711161971092224, -1.5288430452346802, -1.2604914903640747, -0.7066977620124817, -1.568847417831421, -0.3217617869377136, 0.06760153919458389, 0.7073805928230286, 0.6109667420387268, 0.9498923420906067, -0.19351688027381897, -0.49152085185050964, 0.036245472729206085, -1.1049860715866089, 1.3335930109024048, 0.49332156777381897, 2.4480388164520264, -1.7411813735961914, -1.0949265956878662, 0.5481184124946594, -0.19407804310321808, -1.7150237560272217, 1.8314592838287354, 0.917142391204834, 0.19552968442440033, 0.9140870571136475, 1.2615388631820679, -0.2960016429424286, 0.4982365667819977, 0.3128780722618103, 1.314251184463501, 0.4651775360107422, 0.8521171808242798, 0.5252605676651001, -1.1096422672271729, -0.18090566992759705, -0.1706068217754364, -0.15005359053611755, -0.3332882821559906, -0.6175182461738586, -0.028253812342882156, -0.46506598591804504, -0.043304868042469025, 0.8497086763381958, -0.47660261392593384, 1.1130512952804565, 1.8845505714416504, -0.11561821401119232, 0.47449108958244324, 0.09881675243377686, -2.093733549118042, -0.12539725005626678, -1.3138922452926636, 1.595245122909546, -0.0564611442387104, 1.694580078125, -0.784424364566803, -1.5466989278793335, 0.8184369206428528, -1.0634024143218994, -0.4563310742378235, 0.6939913630485535, -0.7881401777267456, 1.4416351318359375, 1.5501832962036133, -0.4633176922798157, -0.6144096255302429, 0.3899451792240143, -0.0275775995105505, -0.2058323472738266, -0.3718257546424866, -0.392203152179718, 0.1646115630865097, -0.22956620156764984, -0.22005106508731842, -0.18962755799293518, 1.4053231477737427, 0.05849615857005119, -1.4443029165267944, 1.093026041984558, -1.5823558568954468, -1.8201626539230347, 1.0121492147445679, -0.12399277836084366, 0.31184375286102295, 0.6876484751701355, 0.004519931506365538, 1.9359413385391235, -1.3787670135498047, -0.056078486144542694, -1.1481748819351196, -1.2296279668807983, 0.2488427460193634, -0.632416844367981, -0.6521012187004089, -1.1115256547927856, -1.0969666242599487, -0.20695152878761292, -0.6429594159126282, -0.9842621088027954, 0.29227104783058167, -1.868325114250183, 0.6920049786567688, 0.19900503754615784, -0.9086755514144897, 0.7564969658851624, 1.5700045824050903, 3.2398574352264404, 0.7351970672607422, -0.4773764908313751, -1.1810864210128784, -2.0763068199157715, -0.508857786655426, 1.1662043333053589, 0.8439779281616211, 0.26637697219848633, -2.304600954055786, 1.4031387567520142, -1.6824499368667603, 0.1943311095237732, 0.9747512936592102, -0.08570536971092224, -0.5253961086273193, -1.14815092086792, -0.09058572351932526, 0.5894501209259033, 0.5970019698143005, 1.0161226987838745, 0.37394511699676514, 1.2558698654174805, -1.0699269771575928, -0.0870170071721077, -2.0380403995513916, 2.7530806064605713, 1.1404224634170532, 0.03349503502249718, -0.5818687677383423, -0.38439086079597473, -0.3760254383087158, 1.351545810699463, -0.6302489042282104, 0.5179042816162109, -1.41119384765625, -0.9864246249198914, 0.26045235991477966, -2.0152645111083984, -1.838512659072876, 0.2569999694824219, -0.05864716321229935, 0.5167398452758789, -0.4072943329811096, 0.5589330792427063, -1.0077975988388062, -0.8242683410644531, 0.5573317408561707, -0.7945144176483154, 0.4345415234565735, 1.858884334564209, 0.40338578820228577, -0.8976978659629822, -0.8471246957778931, 0.2075064331293106, -0.11239518225193024, 0.32375726103782654, 0.1628577709197998, 0.9957496523857117, 1.551896333694458, 1.4495142698287964, 1.0738624334335327, -1.1997621059417725, -0.6706965565681458, 0.2377478927373886, -0.04682210087776184, -0.3096896708011627, 0.3110414743423462, -0.4200987219810486, 1.4501651525497437, -0.8234866261482239, 1.0236772298812866, 1.4813435077667236, -0.07922157645225525, -0.5545201301574707, 1.1945607662200928, 0.3144610822200775, 0.526391863822937, 2.0014686584472656, 2.216111660003662, 1.4534409046173096, 0.11228063702583313, 0.5604839324951172, 0.8110145926475525, 1.5374301671981812, 0.8026045560836792, -1.6133028268814087, 0.4641244411468506, 1.1476904153823853, 1.0486574172973633, -0.7252228260040283, -0.3968222141265869, 0.1022283211350441, -1.337032675743103, 0.19783101975917816, 0.10694502294063568, -0.05887843295931816, -1.6830272674560547, 0.526739776134491, 1.3045628070831299, -0.8679972290992737, -1.228536605834961, -0.27394238114356995, -1.0758384466171265, -0.5029253363609314, 0.30709144473075867, -0.23646193742752075, -0.03543310612440109, 0.014313377439975739, 0.4423326253890991, 1.2277462482452393, 0.2444080412387848, -0.1834668517112732, 1.1618226766586304, -0.12491589039564133, 0.025168130174279213, -0.38016802072525024, -0.21187645196914673, -0.1918039321899414, -0.18772639334201813, 0.17633883655071259, 0.616200864315033, 0.10890527069568634, -1.9491877555847168, 0.03289934992790222, 0.5428162813186646, -0.4930962026119232, -0.22827506065368652, -0.4170151352882385, 0.39727285504341125, 0.39199313521385193, 1.2340283393859863, 0.5871832370758057, -0.12861749529838562, -0.7881706953048706, -3.617387294769287, 0.27122920751571655, -1.2980308532714844, -0.18720649182796478, 1.925366997718811, -0.04314194619655609, 1.0275572538375854, 0.11231962591409683, 0.6763219833374023, -0.5247469544410706, -0.821913480758667, 2.4717719554901123, 0.9258788228034973, 1.5979218482971191, -0.44697442650794983, -0.4813072085380554, 0.6037863492965698, 0.3342694938182831, 0.6596133708953857, -0.521774172782898, 0.06563031673431396, -0.43896484375, 1.2297836542129517, 1.0655958652496338, -2.188206672668457, -0.714945912361145, 0.0720834732055664, 0.6825889348983765, 0.2676661014556885, -1.3978639841079712, -1.7525039911270142, 0.727914035320282, -0.23536352813243866, -0.9669831395149231, -0.3597525954246521, 0.5786861181259155, 0.18679435551166534, -0.1679442673921585, -0.2648420035839081, -0.06484653800725937, -0.7882022261619568, 0.22438329458236694, -0.7910192608833313, -2.1596810817718506, -0.6596878170967102, 0.42333051562309265, -1.7497069835662842, 0.136344775557518, 1.3717310428619385, 0.6917685866355896, -0.7213305830955505, 0.5739601254463196, -0.4038779139518738, 0.9012284278869629, 1.8843986988067627, -1.5452076196670532, 0.8108687996864319, 0.5421981811523438, 0.5207122564315796, -0.29078271985054016, -0.13649077713489532, 1.2930136919021606, 1.9063977003097534, -1.1068769693374634, 0.22492700815200806, 0.8251208662986755, 0.42185941338539124, 0.4453803300857544, 0.07503965497016907, -0.05427752807736397, -1.0942325592041016, 1.513369083404541, -1.1100119352340698, -1.8045300245285034, -0.5218731164932251, -0.638654351234436, -0.20759695768356323, 0.22159475088119507, 1.2589447498321533, -0.8832062482833862, -0.1873667687177658, 0.11528511345386505, -1.393264889717102, -0.9072989821434021, -0.06750183552503586, 1.0838907957077026, -0.3395751118659973, -0.9711999893188477, 1.0593369007110596, -2.303457498550415, 1.3440773487091064, 0.3577430546283722, -0.3213289678096771, -0.9422869086265564, 2.262526035308838, 0.5906925201416016, 0.6221092343330383, 1.443423867225647, -1.0315648317337036, -1.7493925094604492, -0.29955577850341797, -0.4063504636287689, 2.043372631072998, 0.2389683574438095, -0.28657957911491394, 1.7859975099563599, -1.3213169574737549, -0.9485487937927246, -0.7305604219436646, -1.1533682346343994, 0.1350610852241516, -0.5896556973457336, 1.4264228343963623, -0.2674098014831543, -0.7739099264144897, -1.2012766599655151, -0.793885350227356, -1.1600780487060547, -0.8313583731651306, -1.3488000631332397, 0.3202272951602936, 0.34765300154685974, 0.2307795286178589, -0.5173925757408142, -0.1141490787267685, 0.8968040943145752, -0.3836418688297272, 0.11289054155349731, -0.048172060400247574, -0.0929899662733078, 0.2957342565059662, -0.7277225852012634, -0.9913939833641052, -0.9919152855873108, -1.3526544570922852, -0.35256585478782654, 0.20774684846401215, -1.2030458450317383, -0.05732232704758644, -1.7743090391159058, -0.2268940508365631, -0.4837436079978943, 2.194620132446289, -0.07725337147712708, -1.043250560760498, 0.41639041900634766, 0.5788331627845764, -0.6715959906578064, 1.72182035446167, -1.246976375579834, -0.9782356023788452, -0.8388015627861023, 1.3538936376571655, 1.3740060329437256, -0.5245395302772522, 0.07351578772068024, 0.032991405576467514, -0.20957686007022858, -0.7174263000488281, -0.19703343510627747, 0.10501531511545181, -1.4858806133270264, 1.0918174982070923, 0.8576308488845825, -0.871183454990387, 0.5173776745796204, 1.319169282913208, 0.678300142288208, 0.48272204399108887, -1.1435322761535645, 0.6162102222442627, 1.5401959419250488, -1.9336037635803223, 0.1897725760936737, -0.2134491205215454, -0.7241358160972595, 0.6398645639419556, -0.5948051810264587, 1.261612057685852, 1.254349946975708, -0.7195889353752136, 1.4785740375518799, -0.14238087832927704, -0.5424322485923767, -1.5828728675842285, 0.5889236927032471, -0.1763879358768463, -1.4181373119354248, -0.7418984770774841, -1.1765661239624023, 0.20571239292621613, -1.8683733940124512, -0.37619227170944214, 2.0665128231048584, 0.7238897085189819, 0.1861937940120697, -1.3403664827346802, -0.32072216272354126, -0.2260081171989441, -0.31824082136154175, 1.4124319553375244, 0.34387609362602234, 0.4646075665950775, -0.7830412983894348, -0.8281449675559998, -0.2127222716808319, -1.029979944229126, -1.0403209924697876, 0.0453401654958725, -1.7585982084274292, 0.523770809173584, 0.0326642170548439, -0.5692111253738403, 0.9881008267402649, 0.5629479885101318, -0.33880722522735596, -0.5209763646125793, 0.770556628704071, 1.7236379384994507, -0.9896518588066101, 0.9902275204658508, 0.007048777770251036, -0.02411292865872383, 1.5537004470825195, -0.15967105329036713, -1.079533338546753, -0.753660261631012, -0.37319818139076233, 1.8777475357055664, -0.5647546052932739, 1.6288166046142578, 0.007782837841659784, -1.6521962881088257, -0.3370421826839447, 0.9538112282752991, -1.2164559364318848, 0.11229376494884491, -0.31890109181404114, -1.011301040649414, 0.17485980689525604, 0.5963749289512634, 1.864890694618225, 0.4076152741909027, 0.9619709849357605, 1.6355311870574951, 0.03804001584649086, 0.4887087345123291, 0.16679371893405914, 0.07834223657846451, -0.935366690158844, -0.7840150594711304, 1.052687406539917, -1.9147461652755737, 0.5768793821334839, -0.11958231031894684, 1.468553066253662, 0.7577335238456726, -0.9657408595085144, -0.16422852873802185, 0.20020586252212524, -0.5387738943099976, -1.5044935941696167, 1.6437848806381226, 0.7458634376525879, -1.2216485738754272, 0.1754896491765976, 0.5669531226158142, -1.4448773860931396, 1.6469441652297974, -0.857674241065979, -1.2669928073883057, -0.2714657485485077, -0.45673203468322754, -0.39669671654701233, 0.19226083159446716, -0.7527214884757996, -0.16523370146751404, 0.5233157873153687, 0.2940595746040344, -0.32210609316825867, -0.7747213840484619, -1.0077145099639893, 1.3715057373046875, 0.21969448029994965, -0.757430374622345, 0.7930744886398315, 0.1520727127790451, -1.049241304397583, -0.7505577802658081, -0.18368254601955414, 0.7917050123214722, 1.6722159385681152, 0.9834105968475342, -0.5811762809753418, -1.513169527053833, 0.7399059534072876, -0.07121817022562027, 0.5465868711471558, 0.11397664248943329, -1.285573959350586, 0.1291157603263855, -0.3189244568347931, 1.6505693197250366, -0.38131245970726013, -2.1247124671936035, -0.726045548915863, -0.5840892195701599, 0.9712029099464417, -1.9254816770553589, -0.14798586070537567, -0.6990912556648254, -0.6721115112304688, 0.6918473839759827, -2.1271653175354004, -0.03887222334742546, -1.46125066280365, -1.164393424987793, -0.0812525600194931, -0.30553337931632996, 0.1349560171365738, -0.533467710018158, 1.9248063564300537, -0.47010788321495056, 0.820398211479187, -0.30240288376808167, 0.48279422521591187, 0.2487601488828659, 1.031815528869629, -0.3037285804748535, -0.08435027301311493, -0.6935011744499207, 0.2695605754852295, 0.4946092367172241, 0.09122876822948456, -0.39400866627693176, -0.046161867678165436, -0.5909029841423035, -0.835883617401123, -0.16708241403102875, 1.0403119325637817, -0.7084073424339294, 0.10021958500146866, 0.6099466681480408, -0.35827940702438354, -0.4018993377685547, 0.2931307852268219, -1.0234586000442505, 0.022464299574494362, 0.8104127049446106, -0.47969111800193787, -0.841400682926178, 1.3297672271728516, 1.8185696601867676, 0.9517743587493896, 1.7784829139709473, -2.3950817584991455, -0.1349865198135376, -0.721397340297699, 0.25569361448287964, 1.833199381828308, -0.4706549644470215, 0.33752137422561646, 1.0190908908843994, 0.012824214063584805, -0.5779613852500916, 0.31452110409736633, -0.08548677712678909, -0.2864552438259125, -1.269241213798523, -0.35177358984947205, -2.1961050033569336, 0.6472300291061401, 0.6243272423744202, -0.06561341881752014, 0.15358661115169525, -3.1011836528778076, 1.4490662813186646, -2.364887237548828, -1.4864873886108398, 0.08477664738893509, -0.06598901003599167, -2.0739474296569824, 1.674471378326416, 1.2670687437057495, -2.2191057205200195, -0.17181496322155, -0.1726989448070526, 0.42978304624557495, 0.4241587817668915, 0.12200306355953217, -0.23542380332946777, 0.08089304715394974, -0.720802366733551, 0.14272429049015045, 0.05502020940184593, -0.7214915156364441, 1.421994924545288, -1.7172988653182983, 1.3225603103637695, 0.5539754033088684, 0.062188711017370224, -1.167039394378662, 0.6983781456947327, -0.23596876859664917, 0.4033481478691101, -0.06502393633127213, 1.5963689088821411, 0.286761999130249, -0.6658011078834534, -2.4175760746002197, -0.5957497358322144, 0.6028379201889038, 0.5613420009613037, 0.39526763558387756, 0.03805744647979736, 0.7231016755104065, 1.0470761060714722, 1.1173210144042969, 1.0044682025909424, -0.25085389614105225, 0.5058062672615051, 0.6975590586662292, -0.3048182427883148, 0.6860743761062622, 0.2002810686826706, -0.2128472924232483, -0.6960904598236084, 0.0366070456802845, -0.5824602246284485, 1.335771918296814, -0.20410992205142975, -0.4719424843788147, -0.03470131382346153, -0.9372787475585938, -0.1296912431716919, -0.9401450157165527, 0.44770199060440063, 1.500241994857788, 0.1759500950574875, -1.10796320438385, 0.4113883078098297, 1.1981004476547241, 0.3733484745025635, 1.3930758237838745, -1.1508795022964478, -0.8873986005783081, -0.737862229347229, -0.25122594833374023, -0.15513744950294495, -1.3505727052688599, 0.7530462145805359, 0.405732125043869, -0.2625615894794464, 0.2952021062374115, 0.412397176027298, -1.5304462909698486, -0.5876151919364929, 0.3056183159351349, 0.8792082667350769, -0.9685689806938171, 0.7803810834884644, -0.03892146795988083, 0.6113944053649902, -0.07850779592990875, 0.777001678943634, 0.2609036862850189, 0.8398106098175049, 1.0601614713668823, 0.5956853628158569, -0.4310586452484131, 0.18828855454921722, 1.9506555795669556, 0.23327316343784332, -1.7672741413116455, -0.8643211126327515, -0.8247837424278259, -1.189530849456787, 0.23530693352222443, -0.3191748857498169, -1.1682218313217163, -1.212298035621643, -1.6095067262649536, -0.17361824214458466, -0.10690584778785706, 0.4370848536491394, 0.6982705593109131, 0.3954932689666748, -0.6702155470848083, -1.0904991626739502, -0.2717520296573639, 0.26927870512008667, -1.5833362340927124, -0.11527080088853836, 0.5989442467689514, -0.020283866673707962, -0.699356734752655, -0.0260967705398798, -0.30169907212257385, -1.1036407947540283, 1.045511245727539, 0.5855796933174133, 0.016459129750728607, -0.14198777079582214, -1.2145957946777344, 0.19450141489505768, 0.6239405870437622, -0.5684189796447754, -0.864258885383606, -0.44215595722198486, 2.7958099842071533, -0.21250078082084656, 1.1821746826171875, 0.34508293867111206, -0.7104402184486389, -0.15384697914123535, -0.6853764057159424, 1.4622266292572021, -0.6067100763320923, 0.6045346856117249, 0.3566412925720215, 0.666335940361023, 0.3098335266113281, -1.4567934274673462, -1.1398749351501465, -1.0401866436004639, 0.8173472285270691, -0.8089367151260376, 0.36300238966941833, 0.5189744234085083, 0.6973119378089905, -0.7033752202987671, -1.7158502340316772, 3.0413057804107666, 3.077634572982788, -0.5684071779251099, 0.5279405117034912, -1.3747258186340332, 1.2944059371948242, 0.19656182825565338, 0.3644546866416931, 1.4383574724197388, 1.1117491722106934, 0.19506941735744476, 0.09345605224370956, 0.718071460723877, -0.8732951879501343, -0.05266103893518448, -0.8915729522705078, 1.110028862953186, 2.338104248046875, -0.1305508017539978, 0.22832657396793365, -0.5799669027328491, 0.030626138672232628, 2.1245813369750977, 0.6792778372764587, -0.5867804884910583, -0.31654858589172363, 0.039621371775865555, -1.6339927911758423, -0.32350361347198486, 0.8796833753585815, 0.6119994521141052, -1.4170645475387573, 1.4741601943969727, -1.6722798347473145, -1.581460952758789, 0.5638329386711121, -0.9945803284645081, -0.07539588958024979, 0.4470537006855011, -0.04602259397506714, -1.8142720460891724, 0.15538866817951202, -0.9816774725914001, -0.18518975377082825, 0.20970122516155243, 0.5756135582923889, 0.5842741131782532, -0.7302780151367188, 0.38674211502075195, -0.21102091670036316, -0.8204829096794128, -0.28809675574302673, 0.6384959816932678, 0.557863712310791, -0.7613934278488159, 0.577706515789032, -0.02966507524251938, 0.13370995223522186, 0.7169339656829834, 0.4798464775085449, -1.3009101152420044, -0.16871385276317596, -0.29150837659835815, -0.7727640867233276, -0.49936643242836, 0.6779095530509949, -0.06349195539951324, 0.22217631340026855, -0.825387716293335, 0.04895208403468132, 1.1842927932739258, 1.0158674716949463, 0.05858903378248215, 1.1100047826766968, -0.8822021484375, 0.015581324696540833, 0.7482654452323914, -1.160377025604248, -0.3295796513557434, 0.06454586982727051, 0.04560520872473717, -1.5994235277175903, -0.1734752207994461, 0.1913519650697708, -0.8991654515266418, 0.7514253258705139, -1.0048893690109253, -0.8812468647956848, 1.9586228132247925, 0.08107712864875793, 0.5720473527908325, -0.24815359711647034, 0.6290998458862305, -0.5338999629020691, -1.077928900718689, -0.7153191566467285, -1.0856411457061768, -0.21210291981697083, -1.9653335809707642, 0.2043924629688263, -0.6780532598495483, 0.024277430027723312, 1.9060178995132446, -1.165387511253357, 0.28840699791908264, 0.6057165265083313, -0.05768606811761856, -0.9398590922355652, -0.8121572136878967, 0.028291497379541397, 0.9749162197113037, -0.1858663558959961, 1.0908114910125732, 0.5616128444671631, -0.08979294449090958, -0.8173977732658386, -0.4801444113254547, -2.7935800552368164, 0.626972496509552, -0.30540168285369873, -0.31215280294418335, 0.6424647569656372, 0.8567492961883545, 0.26619210839271545, 0.024351684376597404, -0.23060312867164612, 0.24086058139801025, -1.1011985540390015, 0.7213616371154785, 1.5339808464050293, 0.9051164984703064, 0.3860990107059479, -1.2524759769439697, 1.4593013525009155, -0.9355592131614685, 1.1525136232376099, 1.5689878463745117, -0.7570788264274597, -1.3214187622070312, -0.10609849542379379, 2.7524938583374023, 0.09957224130630493, 1.5581837892532349, 1.312389850616455, 1.2463864088058472, -0.6316433548927307, 1.518502950668335, -0.48080241680145264, 0.4190370440483093, -0.6425341367721558, 0.18360283970832825, -0.2210550457239151, -0.05350516736507416, 0.7110717296600342, -1.3012638092041016, 0.08099447935819626, -0.8475316762924194, 0.8311114311218262, -0.0426115021109581, -0.4144410490989685, -0.4770505726337433, 0.367942750453949, -2.275733232498169, -0.5026167631149292, -0.6433461308479309, -1.2987138032913208, -0.5700703859329224, 0.11121074855327606, -0.27307549118995667, 1.2339566946029663, -1.3784780502319336, 1.2816779613494873, -0.29074129462242126, 0.991610586643219, -0.544084906578064, -0.6320759654045105, -0.4342988133430481, -0.9471037983894348, -0.8588390946388245, 0.022183014079928398, -0.6165582537651062, 1.4913580417633057, 0.18798403441905975, 1.7751238346099854, 1.436360478401184, 0.5666991472244263, 1.5965861082077026, 0.32460811734199524, 0.9307617545127869, -0.8970751166343689, 1.0316073894500732, 1.183870553970337, 1.140864372253418, -0.9528834819793701, -0.3206752836704254, -0.9063149690628052, -0.5417190790176392, -1.1005116701126099, 0.6205775737762451, 1.7775717973709106, 0.8450959920883179, 0.7448096871376038, 0.5677599310874939, 1.3842496871948242, -1.0504056215286255, -0.06982135772705078, -1.6015028953552246, -1.547343134880066, -0.3852899968624115, -0.09287109971046448, 0.23224405944347382, 0.10320480912923813, -0.13207942247390747, -0.9379761815071106, 0.83756422996521, -0.1726658195257187, 1.017961859703064, 0.019924143329262733, 0.32520103454589844, 0.41736310720443726, -0.5885652303695679, 0.10192395746707916, -0.46209490299224854, 1.0043028593063354, 0.11147037148475647, 1.4750226736068726, -1.5320191383361816, -0.2255294919013977, 0.5284367203712463, -1.3645859956741333, 0.5876957774162292, 1.5351240634918213, 1.2432159185409546, -1.5496485233306885, 0.981721818447113, 0.6755186319351196, -1.6589136123657227, -0.6073135733604431, -0.22203364968299866, 0.0470610149204731, -0.2569647431373596, 0.7255361676216125, 1.2566726207733154, -0.3362812399864197, -0.8412402272224426, 0.3758904039859772, 0.6904046535491943, -1.107208013534546, -0.051596514880657196, -1.2546883821487427, 0.6840368509292603, -0.043160855770111084, 1.1328119039535522, 1.9077924489974976, -0.1714933067560196, 0.34677618741989136, 0.6895161271095276, 0.5799760818481445, 0.482413113117218, 0.9801653027534485, -0.7453687191009521, 1.098751187324524, 0.3160540461540222, 0.3147042691707611, -0.8308176398277283, -0.952226459980011, 2.2104785442352295, -0.21849453449249268, 0.42304399609565735, -0.945293128490448, -0.0015524579212069511, 0.26326343417167664, -0.5501789450645447, 0.9795194268226624, 0.4609432518482208, 1.5165941715240479, 0.17906798422336578, -0.5455908179283142, 0.3365444839000702, 0.42698732018470764, -0.6363381147384644, 0.5358356237411499, 1.9783991575241089, 1.501053810119629, -0.09048036485910416, 0.0819433256983757, -0.8126391172409058, -0.265925794839859, 0.0998602882027626, 0.1098540872335434, -2.1662070751190186, -0.3488146960735321, -1.8190630674362183, 0.973332405090332, -0.9685099720954895, 0.48294782638549805, -0.7727969288825989, 0.5447918176651001, 0.7312750816345215, -2.7572951316833496, -1.0697975158691406, -0.1642741560935974, -0.19286786019802094, 1.6852678060531616, 0.3495509922504425, 0.8403935432434082, -0.3629450798034668, -0.4122549295425415, -0.5822972655296326, 2.3339953422546387, 0.2747824788093567, -1.6510124206542969, -0.3869518041610718, -0.42711275815963745, -2.032615900039673, -1.8653912544250488, 0.04236694425344467, -0.7940273284912109, 0.9255543351173401, -0.2857827842235565, 0.4811403751373291, 0.19208672642707825, -1.0616209506988525, 0.7016831636428833, 0.6735796332359314, 0.9005953073501587, 1.6854413747787476, 0.24220828711986542, 0.1575329303741455, -0.02087073028087616, 0.30645325779914856, -0.3607155680656433, 0.28897958993911743, -1.096368670463562, -0.7475345730781555, -0.7381695508956909, -0.2688458561897278, -0.7796346545219421, 1.026673436164856, 0.48476511240005493, -1.9573912620544434, 1.0809348821640015, 1.3901550769805908, 0.7226812243461609, -0.010304886847734451, -2.3062965869903564, -0.3135601878166199, 0.9643550515174866, -2.0220561027526855, -0.4374924898147583, 0.023623919114470482, 0.22852283716201782, -1.9421149492263794, -0.28669944405555725, -1.6503779888153076, 0.04140700399875641, 1.7240970134735107, 1.3399020433425903, -0.39531320333480835, -1.1667544841766357, -0.08750206232070923, -0.11618391424417496, -0.589631974697113, -0.3962831199169159, 1.0051389932632446, -1.6664754152297974, -0.0321095734834671, 1.7482786178588867, 0.7792437076568604, -0.13195499777793884, -2.0723700523376465, 0.46873000264167786, 0.23486605286598206, 0.3160681426525116, -0.8774384260177612, 1.7302579879760742, 0.1709667146205902, 0.1269349455833435, -0.023748518899083138, -0.27631527185440063, -3.0736069679260254, 0.18287676572799683, 1.0802069902420044, -0.13423235714435577, -0.10939253866672516, 0.555992603302002, 0.5285136699676514, 0.4186471402645111, 1.9331927299499512, -1.4945220947265625, 0.2624574601650238, 0.952999472618103, 0.6336147785186768, 0.7869037985801697, -2.725895404815674, 0.828336775302887, -0.5547200441360474, -0.27901211380958557, 0.06517236679792404, -0.12304054945707321, 0.5521885752677917, 1.6157432794570923, -0.34838050603866577, -0.8079948425292969, -0.11215682327747345, 0.8762845396995544, -0.2980117201805115, 0.04564126580953598, -0.4637439250946045, 0.3866170644760132, 0.28194889426231384, 0.087895467877388, 0.7109459042549133, 0.6938967704772949, -0.14134572446346283, -0.384687602519989, 0.8584821224212646, -0.938356876373291, -0.7813348770141602, 0.06584212929010391, 0.6458232998847961, -0.3420444428920746, 1.7480697631835938, 1.1967896223068237, 0.6459971070289612, -0.5487722158432007, -0.7377521395683289, 0.4289230406284332, 0.5636457800865173, -0.9398093819618225, -0.18546241521835327, 0.47495609521865845, 1.420473337173462, 0.5971762537956238, -0.06949906051158905, -0.28841522336006165, 1.1993310451507568, 1.6594536304473877, -1.1248300075531006, 0.41512104868888855, -0.12600183486938477, 0.32830944657325745, -1.0548062324523926, -0.1812756359577179, -1.2718511819839478, -1.6822946071624756, 1.1582311391830444, 1.8498203754425049, 0.9402034878730774, -1.0943377017974854, 0.8958924412727356, 0.7515637278556824, 1.9698829650878906, 1.0034801959991455, 0.6764501929283142, 1.7794125080108643, -0.3884594142436981, -0.23803706467151642, -0.8059816956520081, -0.11724083870649338, 1.1618614196777344, 0.4539409577846527, 0.029202032834291458, -1.1983611583709717, -0.39367300271987915, 0.23324957489967346, -0.5387265086174011, 1.5348294973373413, 1.7506531476974487, 1.1240907907485962, -2.049557685852051, 0.6434980630874634, -0.1560470163822174, -0.1457972228527069, -1.043588638305664, -0.6595973968505859, -2.314896583557129, 0.46733173727989197, -0.5582420229911804, 0.21458332240581512, -0.8498745560646057, -0.09388857334852219, -0.6963643431663513, 1.7051721811294556, -0.5245540738105774, 0.9314509630203247, -0.9747203588485718, -0.0321199968457222, -2.4748713970184326, -0.0783558040857315, 0.24973967671394348, -0.3325425088405609, -1.5126612186431885, 0.5861268043518066, -0.4907742142677307, 0.11182387918233871, -0.08055093139410019, -0.18604633212089539, 1.275226354598999, 0.3788183033466339, 2.4290432929992676, 1.687334418296814, -0.925891637802124, 0.19380441308021545, -1.1179715394973755, 0.7349164485931396, 1.4950772523880005, 0.04456824064254761, 0.7770729660987854, -1.0410189628601074, 1.1799423694610596, 1.4395748376846313, -0.1091122254729271, -2.9367737770080566, -0.6279196739196777, -0.11193443089723587, -0.37017518281936646, 1.7020764350891113, 0.20111185312271118, 1.1393376588821411, 0.5222718715667725, -0.6081077456474304, -0.38750338554382324, 0.12621887028217316, -0.507067859172821, 0.21241861581802368, 1.4614498615264893, 0.20493675768375397, 0.11265832185745239, -0.03277835249900818, -1.609499216079712, -0.0724412202835083, -0.7373928427696228, -0.3731304705142975, 1.1697924137115479, 1.0373682975769043, -1.0102242231369019, 0.5891607403755188, -0.7566428780555725, 0.01252166647464037, 0.29894694685935974, -1.1090062856674194, 0.1374373584985733, -0.4674084186553955, 0.8960124850273132, -0.70163494348526, -0.5156983137130737, -0.3302540183067322, 0.03979898616671562, -0.18897570669651031, 0.5553402900695801, -1.385634183883667, -1.0978028774261475, -1.6120272874832153, 0.9491412043571472, 0.2643742859363556, 0.06728647649288177, -1.9108964204788208, -0.9557126760482788, 0.9132925868034363, 0.5694906115531921, -0.4937264919281006, -0.41223666071891785, -0.9616428017616272, 0.4038177728652954, 0.10060293227434158, 0.6878407001495361, -0.8642418384552002, 0.14420628547668457, 0.13297420740127563, 0.67006915807724, 0.39488324522972107, 1.7744132280349731, -0.21533887088298798, 0.40433356165885925, 1.1833828687667847, 1.3126729726791382, 0.0032134708017110825, 1.4012796878814697, 1.2412315607070923, -1.014945387840271, 0.01789620704948902, 1.4950295686721802, 1.298458218574524, -0.032548464834690094, -1.2941356897354126, -0.15865063667297363, 1.0628135204315186, 0.4262271523475647, 1.1409633159637451, -0.3444057106971741, 1.6841078996658325, 0.06315161287784576, 1.1518110036849976, -0.5515915155410767, 1.806504249572754, -0.2843732535839081, -0.2018071711063385, 0.11729826778173447, -1.067155122756958, -0.9338252544403076, 0.03776851296424866, -0.8497658967971802, 0.8151779770851135, 0.22792568802833557, 0.4061807096004486, 0.07349788397550583, -0.52671217918396, 0.10951436311006546, -0.26437002420425415, 1.136926531791687, -2.0198817253112793, -0.18831047415733337, -0.7718431949615479, -1.5749508142471313, -0.729077935218811, -1.4011746644973755, 0.680729329586029, 2.1997010707855225, -1.7100111246109009, 0.8231344223022461, 0.46626317501068115, 0.49988454580307007, 1.2298414707183838, -1.30583655834198, -0.6864515542984009, -0.4094688296318054, 0.5248990058898926, -0.6089389324188232, -2.0582165718078613, 0.3629577159881592, -1.4127142429351807, 0.606453537940979, -0.021247241646051407, -0.37316522002220154, -2.090642213821411, 1.2586239576339722, 0.20136268436908722, -0.47186145186424255, 0.005137079395353794, 0.12082470953464508, 0.1259884387254715, 0.25065141916275024, -2.466522216796875, -1.8954356908798218, 0.2790452241897583, 1.8620809316635132, -0.7469748258590698, -1.1338152885437012, 0.7922651767730713, 1.0057636499404907, -0.22261197865009308, 0.8778788447380066, 0.28592970967292786, 2.0107765197753906, 0.6495341658592224, 0.5726636052131653, 0.5427181124687195, 0.21702270209789276, -1.0196068286895752, -0.3253980278968811, 1.0242375135421753, -1.1690759658813477, 0.850843608379364, -1.073190689086914, 0.9449642896652222, -1.4989804029464722, -0.6503666639328003, 0.11400970816612244], [2.227728843688965, -0.594456672668457, -1.6360218524932861, 0.01033727265894413, 0.6058784127235413, 1.2722523212432861, -1.613493800163269, -0.9304333329200745, 0.40522128343582153, 0.8837583065032959, -0.7689926028251648, -0.5107415914535522, 0.17511402070522308, -1.3607945442199707, 0.1560332179069519, 1.9090474843978882, 0.6028482913970947, -0.49993762373924255, -1.218526005744934, -0.8769170641899109, -0.3284544050693512, -1.9714856147766113, -0.8754786252975464, -0.5035077929496765, -0.38274991512298584, 0.4140172600746155, -0.7341558337211609, 0.06731204688549042, -0.2521916627883911, 0.9087827801704407, -0.4445865750312805, 0.2557053565979004, 0.45492056012153625, -1.3641244173049927, -1.2311230897903442, 0.6901111006736755, 0.2672036588191986, 0.5911229848861694, -0.40655866265296936, -0.10431595891714096, 0.05690367519855499, 0.7914795279502869, -0.0920553207397461, -0.1891993284225464, 0.009403800591826439, -0.4197736382484436, 0.01417990867048502, 0.17170970141887665, -1.1615159511566162, 2.1668200492858887, -1.1700000762939453, 0.565165102481842, 0.830948531627655, 0.4593811333179474, -0.737622082233429, -1.211709976196289, -0.6388779282569885, -0.6406620144844055, 1.0605190992355347, -2.187562942504883, -0.7550708651542664, -1.2795021533966064, 3.0961849689483643, -0.1909128874540329, 0.8129688501358032, 0.16288013756275177, 0.22426874935626984, 0.6064348220825195, -0.13188564777374268, 1.1317205429077148, -0.8762460947036743, -0.44246044754981995, -0.1788979172706604, 1.1654480695724487, 0.00038922668318264186, -0.576698362827301, 1.8991700410842896, -0.23867672681808472, -1.320678949356079, 0.0019178915536031127, 0.19257701933383942, -0.1991920918226242, 1.0228432416915894, -1.2128244638442993, 0.5377184748649597, 0.4961468577384949, -0.2752770185470581, 2.1556828022003174, -0.8667410612106323, 0.7360496520996094, 0.7642020583152771, -1.0096811056137085, 1.99622642993927, -0.1965167075395584, 1.450727939605713, 1.818118929862976, 0.8173791170120239, -0.46685653924942017, 0.6395096182823181, -1.3431023359298706, 0.5297124981880188, -0.019612343981862068, 0.35074788331985474, 0.4979080557823181, 0.45339280366897583, -0.24195212125778198, -0.5285868644714355, 1.390228509902954, 0.40709370374679565, -0.538955807685852, -0.7934951186180115, -0.41794317960739136, -1.7968471050262451, 1.2935899496078491, -1.262298345565796, 2.0454487800598145, -0.34338515996932983, -0.33185333013534546, -2.5947930812835693, -0.9942688345909119, 0.5232052803039551, -0.6283746361732483, 1.4294534921646118, 0.46491268277168274, -0.3204307556152344, -0.26553475856781006, 1.1776567697525024, 0.9054215550422668, 0.5219823122024536, 1.1422677040100098, -1.596397042274475, -0.3717956840991974, -0.9948901534080505, -0.35663098096847534, 0.9085448384284973, 1.0968377590179443, 1.0579310655593872, -1.979985237121582, -1.8265398740768433, 1.133249282836914, 0.0757688358426094, -1.4909223318099976, 0.914927065372467, 0.22765964269638062, -0.37873294949531555, 1.0288101434707642, -0.2771817147731781, -0.5407776832580566, 0.447086364030838, 0.3992687165737152, 0.06103399395942688, -0.820440948009491, -2.1810755729675293, -2.028501510620117, 0.752149760723114, 0.8494946360588074, -0.26588818430900574, 0.659115731716156, -1.1908597946166992, 1.3638290166854858, 0.12488309293985367, 1.1765321493148804, 1.5904910564422607, 1.1647322177886963, 0.7166004180908203, -0.5156911015510559, -0.837152898311615, 0.47530558705329895, -1.108913779258728, 0.08972933888435364, -0.7176771759986877, 2.2690117359161377, -0.38061413168907166, -1.08394193649292, -1.3540940284729004, -0.8932517766952515, 0.8254656791687012, 0.7861961722373962, 0.900970995426178, 1.0125491619110107, -0.32690513134002686, -0.5960830450057983, -0.6247637867927551, 0.23669487237930298, 0.15603885054588318, -0.5629405975341797, 0.9759004712104797, 0.8471384048461914, -1.0493428707122803, -0.6571897864341736, -1.5112320184707642, -0.04518880322575569, -0.9086654782295227, 0.9716458916664124, -0.7325600981712341, 0.80040043592453, -0.1602385938167572, -0.20559187233448029, 0.3906782269477844, 1.646178126335144, -0.42623472213745117, -0.8704590797424316, -1.495111107826233, -0.08228600770235062, -0.4831438958644867, -1.6285696029663086, -0.3914032578468323, -0.5877083539962769, -0.5340840220451355, 1.9388306140899658, 0.6225307583808899, -0.8754388689994812, 0.41549092531204224, 1.753578543663025, 0.5758018493652344, -0.7541114091873169, 0.5877463221549988, -0.8950144052505493, -1.2577033042907715, -0.3105354309082031, 0.4753379821777344, -0.36352112889289856, 0.3974449634552002, 2.1656572818756104, 0.644888162612915, 0.6179059147834778, -0.09998253732919693, -0.7596203088760376, 0.46109041571617126, -0.4717682898044586, -0.650987446308136, 1.534173607826233, -0.883225679397583, -1.6376776695251465, -0.27623531222343445, 1.017895221710205, 1.4227099418640137, -0.3073839843273163, -1.0802243947982788, -1.8730361461639404, 0.5824748277664185, 0.4068453013896942, 1.6425747871398926, 0.20641081035137177, -0.004573950543999672, 1.1000933647155762, -1.5405946969985962, 0.8006302714347839, -0.9446752667427063, 1.1760417222976685, 0.22183199226856232, 0.151176318526268, 0.21128331124782562, -1.257986068725586, 0.6263247728347778, -0.5422035455703735, 0.2382577657699585, -0.5412469506263733, 0.9667385220527649, 0.3199315071105957, 0.7056386470794678, -0.5481807589530945, -0.07007008045911789, -0.32901936769485474, -1.2337816953659058, -1.528651475906372, -0.3597731292247772, 0.5159862041473389, 0.41391721367836, -0.011186552233994007, 1.052912950515747, -0.8868441581726074, -0.466906875371933, -1.4169508218765259, 0.7859978079795837, -0.08792409300804138, 1.2794270515441895, -0.798376202583313, -1.2683789730072021, -0.2968502640724182, 1.7364171743392944, 0.715403139591217, -0.6985923647880554, -0.9687241315841675, -0.9174656271934509, 0.29285669326782227, -0.03180089220404625, -0.30802851915359497, -0.18988947570323944, -0.6087511777877808, 0.6218888759613037, 1.3609758615493774, -0.12730233371257782, 1.6946032047271729, 1.0564645528793335, -0.30387166142463684, 0.8263183832168579, 0.06464263796806335, -0.9176713228225708, 0.5735391974449158, 2.3224477767944336, 1.1473263502120972, -0.19611074030399323, 0.2687157690525055, -0.44388437271118164, 0.5547532439231873, 0.5902261137962341, -2.379157304763794, -2.096298933029175, -0.18273617327213287, -0.9338748455047607, -0.8037663102149963, -0.6089257001876831, 0.29043397307395935, 0.6729826331138611, 1.837728500366211, -1.2202175855636597, -1.7486097812652588, 0.7645239233970642, -0.9488044381141663, 1.2702924013137817, -0.40428999066352844, 0.2342166304588318, 0.2189694344997406, 0.5933409929275513, -2.0304999351501465, -0.17926116287708282, 0.6230360269546509, 0.8947315216064453, 0.6573440432548523, -1.3611650466918945, -0.6110672354698181, -0.386405885219574, 1.102945327758789, 2.0228676795959473, 1.057632327079773, -1.0879570245742798, -0.2805035412311554, -0.5205807685852051, -0.2130870819091797, 1.308782935142517, -0.38773009181022644, -1.3650227785110474, -0.5266615152359009, -0.37574175000190735, -0.7475510835647583, -0.7326042652130127, 1.7047830820083618, -0.32821032404899597, -0.8588159680366516, -1.2504453659057617, 0.8977661728858948, 2.507411241531372, 0.879039466381073, 1.61548912525177, -2.080312490463257, 0.42081359028816223, -0.42122969031333923, 0.12884379923343658, -1.2800548076629639, 0.23323781788349152, 0.07192730158567429, 0.006537769455462694, -0.88307124376297, 0.0019173725740984082, -0.537543773651123, 0.8225504755973816, 0.5414873361587524, 0.38628411293029785, -1.0906639099121094, -1.3371772766113281, -1.5141948461532593, -0.9155194163322449, -0.773146390914917, 0.9280697703361511, 0.8597157001495361, 0.2220865786075592, 0.11041388660669327, 0.46689078211784363, -0.7301484942436218, -1.2499674558639526, 2.0166327953338623, -0.5269697308540344, 0.19927547872066498, -0.9402806162834167, -0.34849604964256287, 0.5153030753135681, 0.7862198352813721, -0.03565843030810356, -0.7597765922546387, -2.532222270965576, -0.11513100564479828, 0.027927884832024574, -0.6389952898025513, 0.3886760473251343, -0.13944362103939056, 0.6029682755470276, -0.007590932305902243, 1.601693868637085, -0.0008153292001225054, 0.557137131690979, -1.1848119497299194, 0.4026106595993042, -0.6233097314834595, 0.8857015371322632, 1.5397547483444214, 1.5657840967178345, -0.9470531344413757, -1.4397414922714233, -0.2627449333667755, 0.22625796496868134, 1.0877331495285034, 0.16511127352714539, -0.39312443137168884, 0.5274899005889893, 1.9424715042114258, -0.2997056543827057, -1.0802950859069824, 0.9055070877075195, -0.7348127961158752, 0.23345747590065002, 0.4891985356807709, 0.033149246126413345, -0.08830618113279343, -0.7678036689758301, 0.1846095770597458, 0.8667103052139282, 1.9267656803131104, -0.14597254991531372, 0.5964037775993347, 0.08492014557123184, -0.8823295831680298, -0.593332827091217, 0.3641047477722168, 0.2116512805223465, -0.6905642151832581, 0.15004441142082214, -0.33902451395988464, -2.2369143962860107, -0.3748447000980377, -0.3175623416900635, 2.0364835262298584, -0.8801792860031128, -1.5701148509979248, -0.16844117641448975, 0.016625775024294853, -0.27782461047172546, -1.6007256507873535, 0.5204352736473083, -0.3876079320907593, -1.1608805656433105, -0.5831559896469116, -0.3569524884223938, -0.17846150696277618, 0.9617093801498413, -0.008485918864607811, 0.7227367162704468, -1.7712377309799194, 2.827924966812134, -1.0937498807907104, -1.2521947622299194, -0.6525694727897644, 1.032607913017273, 0.7642611265182495, 0.5995281934738159, 0.39030641317367554, 1.231201410293579, -1.6529752016067505, -0.2272491157054901, 1.0578943490982056, 0.46008366346359253, -1.574476957321167, 1.082554817199707, 0.6362165212631226, -1.398066759109497, 0.24269652366638184, -0.5022833347320557, 0.5404905080795288, -1.5704371929168701, 2.303790330886841, 0.5920130610466003, 0.5549099445343018, -0.5949574708938599, -0.1981530636548996, 0.8726494908332825, -0.9740999937057495, -2.321108341217041, -1.1263575553894043, 0.6133254766464233, -0.3118613064289093, -0.9715129733085632, -1.2347501516342163, -0.8792787194252014, 0.693757951259613, 1.154615044593811, -0.33183321356773376, -1.4150105714797974, 0.11241161078214645, -0.0660325214266777, 0.4810057580471039, -0.011656242422759533, 2.1046533584594727, -0.328465074300766, 2.146646738052368, -0.029292751103639603, -0.29535195231437683, 0.6773350834846497, 0.4436590075492859, 0.625577986240387, -1.7164278030395508, -0.08724004030227661, 0.6026467084884644, 0.6560589671134949, 1.1389862298965454, -2.173856735229492, -0.21260786056518555, 0.09060412645339966, -0.729793131351471, -1.9543077945709229, -0.013561212457716465, 0.6919138431549072, 0.5672087073326111, 0.8914293050765991, 0.3171897232532501, -0.8377998471260071, 1.3118535280227661, -1.0460448265075684, 0.7928701043128967, 0.17861856520175934, 0.057222429662942886, 0.7531194090843201, -1.5894261598587036, -1.8280175924301147, 1.7137407064437866, -0.5439082980155945, -1.5745896100997925, -1.8911170959472656, -0.5437391996383667, -0.6110656261444092, 0.24338845908641815, -1.0248000621795654, -0.16188126802444458, -0.19975264370441437, -0.6961456537246704, 0.1108027994632721, -1.382804274559021, -0.4299551844596863, 2.7858963012695312, 1.9753917455673218, -1.5705476999282837, 0.4762316644191742, 1.1293202638626099, 0.8174982070922852, -0.32616090774536133, -1.7688469886779785, -1.1141915321350098, 1.1650192737579346, -0.520037829875946, -0.3438318073749542, 2.9023008346557617, -0.3018991947174072, 2.9254307746887207, 1.0965749025344849, -0.8419467210769653, -0.8052286505699158, 2.0556752681732178, 0.039857614785432816, 0.5442707538604736, 0.9570634961128235, 0.3148109018802643, -0.25474855303764343, 0.1644001603126526, 0.03171636909246445, 1.3804601430892944, -0.0461307093501091, -0.18340039253234863, -0.1286575198173523, -0.15302926301956177, -0.8347455859184265, 0.2967185974121094, 1.1641701459884644, 1.45986807346344, 1.0925889015197754, -1.1595538854599, 0.22393843531608582, 0.09974499046802521, 2.313753604888916, -1.0377846956253052, 0.8298234939575195, -0.11473166942596436, 0.5095678567886353, 0.08720576763153076, -0.5066211819648743, -0.5294783115386963, -0.938453197479248, -0.31691500544548035, -0.9642429351806641, 0.5224795937538147, 0.1789073646068573, -0.2318805754184723, -0.6996181607246399, -0.43638285994529724, 0.17716281116008759, 0.9091246724128723, -0.7543541193008423, 0.577618420124054, 0.44361329078674316, 0.8670790195465088, 0.049673423171043396, -0.20624881982803345, -2.140907049179077, -0.4691070020198822, -0.19012124836444855, -0.3260526955127716, -0.7271591424942017, 0.26878002285957336, -0.06218001991510391, 0.7585927248001099, -0.33858439326286316, 0.1414491832256317, -0.18419893085956573, 1.0251233577728271, -0.7471721172332764, 0.9589579701423645, -1.3935415744781494, -0.7298799753189087, -2.4964358806610107, -0.9591605067253113, -0.2540860176086426, 0.5130584239959717, -0.7084808349609375, -0.6821776032447815, 1.0086612701416016, -0.06151703745126724, 0.06416651606559753, -0.021516771987080574, 0.5570030808448792, -0.44415566325187683, 0.8955720067024231, -0.8132234215736389, -3.769148349761963, 0.08603695034980774, 0.7977253794670105, 1.1582441329956055, -1.0506088733673096, 1.1521494388580322, -0.9531825184822083, 0.7629451155662537, -1.1568087339401245, 0.7273638248443604, -2.0453996658325195, -0.5349375009536743, -1.2840404510498047, -1.4158459901809692, -0.2074194997549057, 0.33524084091186523, -0.7834380865097046, -1.6514240503311157, -1.7561546564102173, 1.2009313106536865, 1.8367456197738647, 1.72597074508667, 0.21650642156600952, -0.1567794382572174, 0.5175970196723938, 1.3939944505691528, -1.5931649208068848, 0.5067115426063538, -0.5794609785079956, -1.2624609470367432, 0.0633506178855896, 0.08382884413003922, -0.3738017976284027, 1.540079116821289, 0.9815053939819336, 2.168389320373535, -1.0777239799499512, -0.767381489276886, 0.3959472179412842, -1.0053985118865967, 2.040370464324951, -1.128206729888916, 0.14363883435726166, 0.8006494641304016, -1.013957142829895, 0.6035046577453613, -0.11626552045345306, -0.08917627483606339, 0.39074885845184326, 0.8698720335960388, -1.1383665800094604, 0.9198020100593567, 0.055523306131362915, -0.6518848538398743, -0.05509716644883156, -0.7973939776420593, -0.30712890625, 0.6301471590995789, 0.9596920013427734, -0.6325342655181885, -0.394999623298645, -0.2600175738334656, -0.11825890094041824, 1.469317078590393, -0.17009170353412628, -1.2611538171768188, 2.028972864151001, 0.2979786694049835, -0.567369282245636, 0.319014310836792, 1.2664196491241455, 1.1202549934387207, -0.04593614488840103, 0.5636175274848938, -0.5238314270973206, -0.1010543704032898, 1.2938824892044067, 0.10731121897697449, -1.0504459142684937, 0.6093422174453735, -0.7589715719223022, 0.4501705765724182, -0.015201295726001263, -0.26354509592056274, -0.2709338665008545, -0.4607926607131958, 2.1137542724609375, -0.7390959858894348, -0.3446981608867645, 0.19879810512065887, -0.5521960854530334, 1.4355745315551758, 1.0831037759780884, 0.405439168214798, -0.8431993126869202, -0.3090420067310333, -0.2583686113357544, 0.4960867166519165, 0.618899405002594, -0.4460511803627014, 1.1412309408187866, -1.66211998462677, -0.4106297194957733, 0.649593710899353, -0.2928670048713684, 0.10891936719417572, 0.5224581956863403, -0.5890788435935974, -1.794883131980896, 0.7362629175186157, -0.9445967674255371, -0.35080093145370483, -0.36265307664871216, -0.03914361074566841, -1.1589277982711792, -0.31245774030685425, 0.6291179656982422, 0.4099939167499542, -0.6090352535247803, 0.2547641098499298, -1.1854937076568604, -0.16536164283752441, -1.7069796323776245, -0.7196981906890869, -0.02675013616681099, -0.11788882315158844, -0.7961516380310059, 1.152943730354309, -1.5335631370544434, -1.3112905025482178, -1.5239524841308594, -0.5411781072616577, 0.5824244022369385, -0.03096587210893631, 1.807340383529663, 1.4348089694976807, 1.0600957870483398, 0.3173706829547882, -1.2261583805084229, -0.3158216178417206, 0.07513940334320068, -0.6198452115058899, -1.7641515731811523, 0.7548827528953552, -0.06661618500947952, -0.22893044352531433, 0.31237560510635376, 1.5780161619186401, 0.5574761033058167, -0.22423705458641052, -1.2566494941711426, 0.7391183972358704, 0.5694929361343384, -1.457031011581421, 0.561043918132782, 0.2691912353038788, 0.07517413794994354, 1.0472068786621094, -0.24532203376293182, -0.7265447974205017, -0.16722628474235535, -0.0647340640425682, 1.0186816453933716, -0.5495797991752625, 0.6173151135444641, -0.4650522768497467, 0.8725782036781311, 0.6104639768600464, -0.3133178651332855, -0.6776105761528015, 0.356534868478775, 0.0169597826898098, -1.8649766445159912, 0.5669531226158142, 0.9970762729644775, 0.18382932245731354, 1.0526889562606812, 0.5153636336326599, 1.15109121799469, 0.6508029699325562, 0.6770450472831726, -1.0752884149551392, 0.10001711547374725, 2.456827402114868, -0.9976085424423218, -0.010475222021341324, 2.1313564777374268, -0.5869072079658508, 0.047010838985443115, -0.38525474071502686, 0.9525991082191467, -0.5287922024726868, 0.8530603051185608, -0.6111155152320862, -0.9931241869926453, 1.0978139638900757, -0.7037827372550964, 0.3668927550315857, 0.8033072352409363, 0.6725278496742249, -0.47149884700775146, -1.3628010749816895, 1.129695177078247, -0.2913641035556793, -0.07397692650556564, 0.16493238508701324, -0.2089982032775879, -1.1390618085861206, -0.12020743638277054, -0.5991770625114441, 0.6176028251647949, 1.3161617517471313, -0.5765560269355774, -0.6389161348342896, -1.2050824165344238, -0.5664695501327515, 0.9840139150619507, 0.3660105764865875, 0.05331050977110863, -1.1324756145477295, 0.21022440493106842, 0.4637473523616791, -0.4252602159976959, 0.79928058385849, -0.018086718395352364, -0.17868128418922424, -0.32817134261131287, -2.1746952533721924, -0.04104246944189072, 1.612412452697754, 0.4153522551059723, 1.2276448011398315, -0.2663846015930176, -1.165317177772522, 1.4709049463272095, 0.8044466376304626, 1.1382198333740234, -0.21508190035820007, -0.23902568221092224, -0.21882861852645874, -1.4766851663589478, 0.29491710662841797, 1.567143201828003, 1.185391902923584, -0.20003457367420197, -0.17333580553531647, 1.861536979675293, -0.2031145542860031, -0.9944683909416199, 0.5531023144721985, -0.513241171836853, 1.0052831172943115, 0.0665484219789505, 0.7485841512680054, 0.3842022120952606, -0.8667247295379639, -0.27272066473960876, -0.29514017701148987, -1.9252049922943115, 1.365520715713501, 0.7619444727897644, -0.10848181694746017, -0.1742011159658432, 0.3234328329563141, -0.006785908713936806, -0.2101612538099289, -0.4387155771255493, 1.4908294677734375, -0.9606431126594543, -2.317349672317505, 0.8955048322677612, -1.354427456855774, -1.0946310758590698, 0.1941557675600052, -0.5285502672195435, 0.44081711769104004, -0.08827061206102371, 0.30698853731155396, -0.7046904563903809, -1.0439661741256714, 0.22373852133750916, -0.38000810146331787, 0.6967068314552307, -0.8567549586296082, -0.35902976989746094, -0.6220481991767883, -1.451974868774414, 0.22330592572689056, 0.1570775806903839, -0.6354273557662964, 0.2552083432674408, -1.6766375303268433, 0.9425561428070068, 0.732445240020752, -1.4484343528747559, -0.8115279078483582, 1.6611961126327515, -1.0981932878494263, 0.42428696155548096, -1.1081876754760742, 0.3195526599884033, -1.1338245868682861, 0.6201571226119995, -0.9193509221076965, -0.28425779938697815, -0.09549305588006973, -0.4234415292739868, -0.01548337284475565, -0.5121285319328308, 1.0373986959457397, -0.7793666124343872, -0.4445144236087799, 1.1389967203140259, -0.09793545305728912, -1.0205961465835571, 1.2688089609146118, -0.06939104199409485, -0.5935285687446594, 0.7384015917778015, 0.43322616815567017, -1.5569599866867065, 0.40428856015205383, 0.5500041246414185, 0.4742923974990845, -0.19765672087669373, -0.3145427703857422, 0.7494780421257019, 1.1788890361785889, -0.5445113778114319, -0.5400488972663879, -0.3150433301925659, 0.8101134896278381, 1.8830915689468384, -0.3718359172344208, 1.1271274089813232, -2.711601734161377, -0.30666643381118774, -2.739795446395874, 1.4169838428497314, 0.5416825413703918, 2.279674530029297, 0.501179039478302, -0.35338467359542847, 0.36330822110176086, 0.12763194739818573, 0.41364458203315735, 1.7843601703643799, 0.8848423957824707, 0.9179019331932068, 1.5744876861572266, -1.3110697269439697, -0.14671270549297333, -2.4631073474884033, 0.47506192326545715, -1.4194505214691162, -1.5565129518508911, -0.19722065329551697, -0.1897699236869812, -0.8474865555763245, -1.1091874837875366, -1.2909246683120728, -0.05462726578116417, 0.9138861298561096, -1.0731905698776245, -1.8599270582199097, 1.8917981386184692, -0.3960988223552704, 0.1505742073059082, -0.6193978786468506, -0.039218030869960785, -0.07944542914628983, -0.02818235009908676, 0.7933972477912903, -0.9223348498344421, -0.06077440455555916, -1.4945118427276611, -1.5855109691619873, 1.9725228548049927, -1.5228660106658936, 0.9419689774513245, 1.2687032222747803, -1.6886197328567505, -0.5394688844680786, -0.9694200158119202, -1.0438281297683716, -0.378695547580719, 0.4938809275627136, 0.4343707859516144, -0.7255061268806458, 1.535204529762268, 0.6844797134399414, 1.2986969947814941, -1.5475422143936157, 1.0425190925598145, -0.873316764831543, -0.44498735666275024, -0.6179304718971252, 0.09146799892187119, -0.5376130938529968, 1.2461165189743042, 0.5159271359443665, 2.102386713027954, -0.6898260116577148, 2.074852705001831, 0.22565311193466187, -0.5342649817466736, -0.6682577133178711, 1.1058181524276733, 1.464004635810852, 1.1549770832061768, -0.005823296960443258, 0.12255322188138962, -0.6661877632141113, -0.7335836291313171, -0.24032986164093018, 1.0223405361175537, -1.066299557685852, -0.19376246631145477, 0.6792418956756592, -0.5770694613456726, -0.022329987958073616, 0.16655345261096954, -0.8726003766059875, 0.7023163437843323, 0.40381935238838196, -2.4046554565429688, 2.5251171588897705, 0.8450469970703125, -1.5338577032089233, -0.0032890615984797478, -0.23152810335159302, -0.15126949548721313, -1.5453696250915527, -2.007420301437378, -0.2269008308649063, -1.4146292209625244, 1.6295726299285889, 0.06475480645895004, -0.7307729721069336, -0.011031110771000385, 1.6889957189559937, -0.17892684042453766, 0.38010963797569275, 0.1174987331032753, -1.0361136198043823, 0.45574358105659485, 0.8942372798919678, 0.48084700107574463, -0.5484139323234558, 0.6444198489189148, 1.4613687992095947, -1.39980947971344, 2.116081953048706, -0.27720868587493896, -0.8380734324455261, -0.4664979577064514, -0.05709049105644226, 0.41619449853897095, 0.5400925874710083, 0.22765910625457764, -0.7594557404518127, 1.9876495599746704, 0.44772306084632874, 0.1554216593503952, 1.5189201831817627, 0.8434101939201355, 1.0086320638656616, 0.08655183762311935, -0.5599982738494873, 0.6943009495735168, -0.09241825342178345, 0.5562340617179871, 0.21651773154735565, -0.7314547300338745, 1.1079888343811035, 0.6089310646057129, 0.5516733527183533, 1.9646135568618774, 1.3110977411270142, 1.7208914756774902, 0.37393099069595337, -2.8725244998931885, -0.7527360320091248, -1.2879064083099365, 2.1112518310546875, 1.0277591943740845, 1.1553932428359985, 0.1983138769865036, -0.3260931670665741, -0.6931740641593933, 0.017353925853967667, -0.6914182901382446, 0.3263002932071686, -0.22169624269008636, -0.887717068195343, 1.3485450744628906, -1.330214500427246, 0.06771843135356903, 0.8528504967689514, 0.6935769319534302, -0.07683948427438736, 0.5094922184944153, -1.0522067546844482, 0.34080275893211365, 1.1146519184112549, 1.2055944204330444, 0.7823435664176941, -1.6638702154159546, -1.2385767698287964, -0.5867284536361694, -0.2651672959327698, -0.1537424921989441, -0.4605206847190857, 0.779913067817688, -1.8992648124694824, 0.020939139649271965, -0.3477022349834442, -2.3493454456329346, -2.3505473136901855, 1.3280595541000366, 0.07631141692399979, -2.3647170066833496, -1.2810347080230713, -1.0773670673370361, 0.04884360730648041, 0.786395251750946, -2.7440690994262695, 2.5887222290039062, -0.3924795389175415, 0.46520835161209106, -0.8522056937217712, -0.5159087777137756, -0.25420013070106506, -1.4750269651412964, 0.5793206095695496, 0.6359100341796875, -1.7842583656311035, -0.15714795887470245, -0.974627673625946, 0.30856853723526, -1.201986312866211, -1.2820839881896973, -0.6365513205528259, 0.40433579683303833, 1.1488388776779175, 0.26015445590019226, -0.6846853494644165, 0.3260953426361084, 2.0848793983459473, 1.0076427459716797, -1.2188152074813843, 0.13159076869487762, 0.9393084645271301, 0.3407789468765259, 0.2022615522146225, 2.001934766769409, 1.4628576040267944, 0.43881943821907043, 1.0401581525802612, -0.030423514544963837, 0.5916914939880371, -0.639337420463562, -1.9978430271148682, -1.5977773666381836, -1.4619817733764648, -0.310398131608963, -2.733051300048828, -0.10392767935991287, -1.7044566869735718, 1.711926817893982, -0.054665468633174896, -1.0760360956192017, -0.7851057052612305, -0.6200066208839417, 0.857582151889801, 0.1802178919315338, -0.27272501587867737, -1.4943556785583496, 0.12135165184736252, 1.4750217199325562, -0.14886076748371124, 0.4310881495475769, 0.09597894549369812, -0.3350519835948944, 0.767899215221405, -0.2786668539047241, -1.7197514772415161, 0.2816845774650574, 0.8617615103721619, 0.14042925834655762, 2.434847831726074, -1.4903281927108765, 0.1851436346769333, 0.9880306720733643, -0.23521308600902557, 1.1400598287582397, 1.1574877500534058, 1.590657353401184, 0.7023686766624451, 0.11215520650148392, -1.188288688659668, -1.4893131256103516, 0.8133365511894226, 0.35442033410072327, -0.8323390483856201, -0.2753114700317383, -0.7571491003036499, -0.44430238008499146, 0.6952559351921082, -0.0044058384373784065, 0.08920372277498245, -0.8716843128204346, -0.4061265289783478, 0.6988381743431091, 0.5483016967773438, -0.954864501953125, 1.1481297016143799, 0.9798718690872192, -0.6192899942398071, 1.1296812295913696, 1.0301258563995361, -0.47781768441200256, -0.9509125351905823, -0.32302501797676086, 0.16877514123916626, -0.8182771801948547, 0.48602184653282166, -0.6709010601043701, -0.8601142764091492, 0.5442304015159607, 0.11761109530925751, -0.9241413474082947, 1.3419911861419678, -0.38308045268058777, 0.07830559462308884, 2.2191169261932373, 1.1570558547973633, 2.6199326515197754, -1.6641579866409302, -0.5442887544631958, 2.6381654739379883, -1.1113966703414917, 1.4291222095489502, -1.190955400466919, -0.3068103492259979, -0.1756235659122467, 1.571515679359436, -0.40803471207618713, -1.8282772302627563, 0.5605933666229248, 0.11950155347585678, 0.10650335997343063, -0.3969320058822632, 2.1007444858551025, -0.1177562028169632, 1.5943787097930908, -1.2584178447723389, -0.024659913033246994, 0.39879173040390015, 0.12232671678066254, -1.377160668373108, 0.6164165735244751, 2.131288766860962, -1.0887494087219238, -0.5595976710319519, -0.2081356793642044, -0.7208467125892639, -0.9844849109649658, 0.32871946692466736, -0.8114562034606934, 0.6560003161430359, 0.9638007283210754, 0.36524859070777893, 0.3591587245464325, 1.3357868194580078, 1.8735145330429077, -0.7571697235107422, -0.2565491199493408, 1.773593544960022, 0.14528976380825043, 0.42859330773353577, 0.6068278551101685, 2.2498719692230225, -0.6799027919769287, -0.47831305861473083, -0.812348484992981, -0.33039218187332153, -0.7013610601425171, 3.8891379833221436, -0.077255979180336, 0.5477426052093506, -0.6949324011802673, 0.3564966320991516, -1.9549974203109741, 0.9612599611282349, -0.238722026348114, 0.6680295467376709, -0.03326128423213959, -0.6372409462928772, -0.8438524007797241, -0.20903600752353668, 0.051930345594882965, 0.17677289247512817, 1.1896191835403442, 1.269782304763794, -0.49040743708610535, -1.6183222532272339, -0.9902003407478333, 1.295072078704834, 1.198826789855957, -0.3265116512775421, 0.6078280210494995, 0.30659812688827515, 1.180753469467163, 0.6615541577339172, 0.5492538809776306, -0.43250688910484314, -0.5446687936782837, -0.07460886985063553, 0.599149227142334, 0.7490106821060181, 2.0530760288238525, -0.29321005940437317, 1.137206792831421, 0.2548607289791107, 0.5976997017860413, 0.265018105506897, -0.5123420357704163, -0.3537907600402832, 1.2043522596359253, -0.7235206365585327, 0.3950267732143402, 1.17649507522583, -0.17312966287136078, -0.6900159120559692, 0.7419759631156921, -0.5453373789787292, 0.4579102396965027, 1.7244845628738403, 1.8352789878845215, -1.0320096015930176, -0.6098647713661194, -0.3568960726261139, 0.39247408509254456, 0.3678975999355316, -0.5763018131256104, 0.25754231214523315, -0.0380619578063488, 1.7008663415908813, 0.918749213218689, -0.8052471876144409, -1.1624367237091064, 0.10934136062860489, 1.7816331386566162, -0.7628541588783264, 0.07173646986484528, -0.6638163328170776, -0.23068492114543915, -1.1138379573822021, 0.9336189031600952, -2.313478946685791, -1.6242095232009888, -0.601579487323761, -0.9926574230194092, 0.35876330733299255, -0.3928690254688263, -1.2754319906234741, -0.18179760873317719, 2.1155765056610107, -0.1337282657623291, -0.8118004202842712, -1.310957431793213, 0.7078937292098999, -0.8193051815032959, 0.1929323822259903, 0.43823179602622986, 0.6084033846855164, -1.0656646490097046, 0.7156628966331482, -1.827894926071167, -0.9547531604766846, 0.30707359313964844, 0.28056764602661133, -0.799791693687439, 0.9550822973251343, -1.3039878606796265, 0.12004489451646805, -0.9253025650978088, 0.21031156182289124, 0.15595261752605438, -0.49751704931259155, 1.1399277448654175, -0.4333595931529999, 0.0916212722659111, 0.0210221316665411, -0.9099895358085632, 0.3340310752391815, -2.088000774383545, 0.3291236460208893, 1.3109095096588135, 0.5730748772621155, -0.8465850353240967, 1.6504892110824585, 0.247574120759964, 1.631681203842163, 0.7812767624855042, 0.8398672342300415, -0.747155487537384, -0.5796326398849487, -0.8048720955848694, 0.2848968505859375, -0.03646061569452286, 1.3887715339660645, -1.1249964237213135, -0.3455832898616791, 1.161017656326294, 0.5831863284111023, 0.10145207494497299, -0.09917005896568298, -0.29658523201942444, -2.209129571914673, 1.1033260822296143, -1.8761711120605469, 0.5827754735946655, -0.8659931421279907, 1.029018521308899, 0.1297936737537384, 1.2150146961212158, 2.112567186355591, 0.4860609769821167, -0.15590186417102814, 1.2329832315444946, 0.33088594675064087, 0.20541124045848846, 0.8010583519935608, 0.44542476534843445, -0.9684536457061768, 0.4897178113460541, -1.6312259435653687, -2.2783398628234863, -1.3436301946640015, -0.8689601421356201, 0.6912322044372559, 1.217828392982483, -1.1474425792694092, -0.28961899876594543, 0.03112971968948841, -0.12079032510519028, -0.2537623345851898, 0.1936928927898407, -0.5999897718429565, -0.8626559376716614, 0.2410942167043686, 0.0808618813753128, -0.44148826599121094, 2.602888345718384, -1.281576156616211, -0.3998155891895294, 0.92170250415802, 0.02712377719581127, -0.2023942768573761, 0.8190248012542725, 0.09658932685852051, 0.6345932483673096, 1.0883362293243408, 0.4633735716342926, 0.18764561414718628, -2.6824839115142822, 0.4766664206981659, 0.7008860111236572, 0.10142811387777328, 0.6193925142288208, 0.6511116623878479, -1.30706787109375, 0.4686916768550873, 1.7420670986175537, 0.5610109567642212, 0.5695160627365112, 1.7021673917770386, 1.5804023742675781, 0.6006232500076294, -0.1428992599248886, -0.8789802193641663, -0.662839412689209, -0.04452676698565483, 0.874279797077179, 1.88511323928833, -1.5136317014694214, 0.0067143221385777, 0.5217398405075073, 1.5052858591079712, -0.394339919090271, -0.6093534827232361, -0.2021532952785492, 1.2461367845535278, 0.07620763778686523, 1.1292847394943237, 0.014037641696631908, -0.6612173318862915, -1.4781960248947144, 0.37079834938049316, -0.17763042449951172, 0.7618972659111023, -0.8672869205474854, 0.9828360080718994, 1.8108595609664917, 0.8622162938117981, 0.38489052653312683, 0.8966063857078552, -0.5395310521125793, -2.2637667655944824, 0.36796292662620544, 0.6809309720993042, 0.8733647465705872, -0.6479933857917786, -1.307450771331787, 0.5014715790748596, -0.8566091060638428, -0.1389816403388977, -1.2682907581329346, -1.121256709098816, -0.49874454736709595, -0.9873194694519043, -0.13378740847110748, 2.1562113761901855, -2.097517728805542, -2.247389554977417, 0.8142460584640503, 0.5761619210243225, -0.4131724238395691, 1.352060317993164, 0.8700831532478333, -0.36017975211143494, -1.9951889514923096, 0.8711674809455872, -0.5450434684753418, 1.398695707321167, 0.46148431301116943, 0.8673599362373352, 0.0036582876928150654, -2.0633625984191895, 1.0936851501464844, -2.724428653717041, 0.9979864358901978, -1.0621674060821533, -2.4168615341186523, 0.38141486048698425, 0.6527882814407349, -0.05748721957206726, -1.3512729406356812, 1.4464374780654907, 1.0668162107467651, -0.6656081080436707, -0.42231035232543945, -2.166759490966797, 0.6199737787246704, -0.643523633480072, -1.4599961042404175, 0.9711490869522095, -0.4136950373649597, 1.050779104232788, 1.0990707874298096, -1.3107595443725586, -0.8950912952423096, 0.8481273055076599, -0.9164186716079712, 1.8865501880645752, -1.2475011348724365, 0.0011978673283010721, -0.23122669756412506, 0.6654428839683533, 0.8591744303703308, -0.7965437173843384, -1.3894331455230713, 2.187243938446045, 1.7507728338241577, 0.0424656942486763, -0.5349182486534119, 2.3069067001342773, -0.4515162706375122, -1.535040259361267, 1.528317928314209, 1.0208406448364258, 0.2919439375400543, -0.5835195779800415, 2.46081280708313, -0.8785043358802795, 1.1375552415847778, 0.5776615738868713, 1.0498127937316895, -1.1158239841461182, 0.32815054059028625, 1.2333927154541016, 0.40760114789009094, -0.16255499422550201, -0.15151798725128174, -0.5448840260505676, -0.02698780596256256, -0.6180872321128845, -0.6838595271110535, 2.059542179107666, -1.8136268854141235, 0.6744121313095093, 0.0946316048502922, 0.09882227331399918, -1.00193190574646, 0.8045036196708679, -0.08547405898571014, -0.09968864917755127, 0.9755792617797852, 0.015942225232720375, -0.6260850429534912, -0.9440344572067261, 0.7047659158706665, 0.15902774035930634, -0.13006441295146942, 1.1312038898468018, 0.6706684231758118, 0.5963314175605774, 0.20225182175636292, 0.469752699136734, 1.2032479047775269, -1.2378344535827637, 1.531668782234192, 0.6430700421333313, 0.6456536650657654, -0.6313560605049133, 1.6527537107467651, 1.0754483938217163, -0.8513507843017578, -1.0990958213806152, -1.571963906288147, 0.6560389995574951, 0.15334288775920868, -1.0246567726135254, 0.05147101730108261, 0.025905825197696686, -0.6694945096969604, -0.7063034772872925, 0.41013890504837036, -0.9388677477836609, -2.0305583477020264, 1.9816237688064575, 0.5691819787025452, -1.241508960723877, -1.7116219997406006, -0.7195380330085754, 1.4069771766662598, 2.7707417011260986, 1.5345567464828491, 0.8651313185691833, 0.6238429546356201, 0.6955992579460144, -1.9325103759765625, 0.30861738324165344, -0.08031698316335678, 0.3965892791748047, 2.8004822731018066, -0.5979011058807373, 1.5402131080627441, 1.1055033206939697, 0.2133806049823761, -1.4833258390426636, -1.1472394466400146, 0.47752541303634644, -1.0991657972335815, 0.20942378044128418, -0.7040731906890869, -0.33857783675193787, 1.6649014949798584, -2.2995290756225586, -1.1310033798217773, -1.7828015089035034, 0.24349743127822876, -0.10927475243806839, -0.427663117647171, -0.4200739860534668, -1.6759964227676392, 0.7824452519416809, 0.19960352778434753, 1.4677000045776367, 0.1346067637205124, 0.4228014647960663, -0.96903395652771, 0.0898280143737793, 0.6273913383483887, 0.2060662806034088, -1.1371402740478516, -0.25891467928886414, 1.023047685623169, -1.7916172742843628, -0.6667864918708801, 0.6643476486206055, 0.09545190632343292, 0.32931193709373474, -1.1340925693511963, 0.053640954196453094, 0.14597605168819427, 0.4469870626926422, -0.15375752747058868, 1.8825569152832031, -1.5715479850769043, -0.47223201394081116, -0.6187461614608765, 0.8763781189918518, 0.8855463266372681, 1.1168889999389648, -1.649385690689087, 0.1667204052209854, 0.25152599811553955, -0.6548492908477783, -0.1914714276790619, -0.40092238783836365, 0.6863117814064026, -2.2037203311920166, -0.1804015040397644, -1.5307140350341797, -1.6621074676513672, -1.190797209739685, -1.12062668800354, -0.4683392345905304, 0.1347586214542389, 0.03581499308347702, 0.09271246194839478, -0.2878882884979248, 2.0587737560272217, -0.7587159872055054, -0.7715136408805847, 0.791970431804657, 1.0855752229690552, 1.1083881855010986, -0.3220204710960388, -0.7225281596183777, -1.3701947927474976, -0.7316072583198547, -0.42374783754348755, 0.7993760108947754, -1.0294269323349, 0.5299025177955627, -0.13062864542007446, 0.46738675236701965, 0.3378824293613434, -0.8044151663780212, -0.045132700353860855, -1.1904057264328003, 1.0383830070495605, 0.12614868581295013, -0.34124457836151123, -0.7996776700019836, -0.18694674968719482, 1.837437629699707, 0.020703734830021858, 2.0840160846710205, -0.6625919342041016, -0.7364640235900879, 1.3620715141296387, -0.4411814212799072, 1.0509426593780518, -0.9133602976799011, -1.0010408163070679, 1.206752061843872, -1.6900050640106201, -0.34799924492836, -0.08415817469358444, -0.006092578172683716, 1.4248539209365845, 0.6286532878875732, 1.2210255861282349, -0.7622475028038025, -0.3109003007411957, -0.7761910557746887, 0.24319706857204437, -1.0251743793487549, -0.6380512714385986, -1.0216337442398071, 0.4750857651233673, 0.7710815072059631, 1.117435336112976, 0.5878426432609558, -0.31466424465179443, 0.5946364402770996, -0.5495247840881348, 0.14993833005428314, -0.23823130130767822, 0.22363755106925964, 1.9193990230560303, 0.972895622253418, -0.7670964598655701, 1.053362488746643, 0.9486891031265259, 1.4945435523986816, 0.07520691305398941, -1.1790107488632202, 0.38837894797325134, -0.05314703658223152, -1.0977674722671509, 0.14629852771759033, -0.5198960304260254, 0.6021578311920166, -0.45549046993255615, -0.441135972738266, -0.49476227164268494, -0.8388869166374207, 0.769623875617981, -0.2727016508579254, 0.04252435266971588, -0.4271065592765808, 1.5111923217773438, 0.48715996742248535, -0.5055793523788452, -1.2481054067611694, 0.6629807949066162, 0.9937307238578796, 1.414780616760254, -2.2330830097198486, -1.9256383180618286, 0.7010968923568726, 0.0016156523488461971, -1.6665029525756836, 2.4814083576202393, 2.2752416133880615, -0.7343616485595703, 1.002881407737732, 0.04614584520459175, -0.8536256551742554, 1.031413197517395, 0.8909193277359009, 0.13899651169776917, -0.5123183727264404, 1.0790603160858154, -1.33847975730896, -0.403237909078598, 0.7503138780593872, -1.912377953529358, 0.41086581349372864, -1.1530230045318604, -1.3965693712234497, -0.9983375072479248, 0.2287648469209671, 0.3405646085739136, -0.4609195291996002, -1.0413029193878174, 1.3508858680725098, 1.813390851020813, 1.013217806816101, -0.21725308895111084, 0.5241938829421997, -0.9918854236602783, 2.794811248779297, 0.32751116156578064, -1.0636193752288818, -0.2952287793159485, -0.8054302930831909, 0.6727052927017212, -0.6239464282989502, -0.19545526802539825, 0.1101541742682457, 1.1796094179153442, -0.6838213205337524, -0.17175355553627014, 1.627748727798462, 1.198983907699585, 0.061837512999773026, -0.767867922782898, 0.3695352375507355, 0.1360786259174347, -0.2951570749282837, 0.686501145362854, -0.5966607332229614, -1.544372797012329, 0.9036355018615723, 0.11563847959041595, 0.9678382277488708, -1.3231532573699951, -0.850212574005127, 0.48038533329963684, -0.788699209690094, 0.2044866383075714, 0.5005291700363159, 0.17776387929916382, -0.582295298576355, -0.7422367334365845, 0.44963976740837097, -0.7022908329963684, 1.0282478332519531, -1.3666058778762817, -0.005426282063126564, -0.8459703326225281, 1.0420957803726196, 0.8730319142341614, 1.2076914310455322, 0.3441290557384491, -2.854663848876953, -0.1600487232208252, 0.38961678743362427, 0.33766788244247437, 0.5477612018585205, 0.055199556052684784, 0.07389318943023682, 0.7555682063102722, 2.4919018745422363, -2.2590949535369873, 1.8796297311782837, -0.13438746333122253, -0.1208224818110466, -0.6526229977607727, -1.0408685207366943, -0.830207109451294, 1.0843870639801025, 0.19278299808502197, 1.4012876749038696, -2.059072256088257, 0.33148741722106934, -1.1063268184661865, -0.26603448390960693, -0.8041426539421082, -0.34810835123062134, -0.17971165478229523, 1.088820219039917, 0.18389558792114258, -0.1496364325284958, 0.27568739652633667, 0.8442031741142273, -0.49619725346565247, -1.0570751428604126, 1.372680902481079, 0.5340293645858765, 0.7403413653373718, -0.02596951276063919, 1.0561319589614868, -0.2635742127895355, -1.4132903814315796, -0.3712063431739807, -0.0190983135253191, -1.2864644527435303, 0.4305790662765503, 0.22178353369235992, -0.6586562991142273, 0.13464081287384033, -0.2254878431558609, -0.5542158484458923, 1.093994379043579, -0.7279874682426453, -0.7165408730506897, 0.7251472473144531, 0.521647036075592, 0.8776508569717407, -1.3763782978057861, -1.6707701683044434, 0.47109493613243103, -0.2111862748861313, -1.0220844745635986, -0.24136964976787567, -0.9266330599784851, -1.0775740146636963, -0.02351153828203678, -0.19614724814891815, 0.5502560138702393, 1.2004272937774658, -0.6313189268112183, -1.2680935859680176, 0.30451640486717224, 0.4012680947780609, -0.9138349890708923, 0.5592321753501892, -0.06789842993021011, 0.0070904274471104145, 1.3528504371643066, 0.15422987937927246, -1.674165964126587, 0.7634027600288391, 0.33305075764656067, -0.4489854574203491, 1.3789886236190796, 1.3639030456542969, 1.2635148763656616, -0.3669739067554474, 0.5461123585700989, 0.6974441409111023, -1.248623013496399, -0.7604804635047913, -0.564765453338623, -0.3345358371734619, -0.44426092505455017, -0.7339052557945251, -0.3672330677509308, -1.1985772848129272, 2.6355626583099365, -1.4722182750701904, 0.7632902264595032, -0.7390640377998352, -0.3545895218849182, -0.4723004996776581, 1.1596970558166504, -1.999186635017395, 0.7443314790725708, 0.4629242718219757, 0.647792637348175, 2.314237117767334, 0.39134836196899414, -0.01980104111135006, -1.7859059572219849, -0.15190176665782928, 0.7992327213287354, 0.5933228135108948, -0.9881576895713806, -1.5971688032150269, 0.8783847093582153, -0.245750293135643, 0.5705335736274719, 0.8064610958099365, 0.14270161092281342, 1.1746875047683716, 0.060138583183288574, -0.04164888337254524, 0.3282008171081543, -0.3674822747707367, 0.9821448922157288, 0.5796668529510498, -0.8248454928398132, -0.5263649225234985, 0.8721974492073059, -0.19800624251365662, 0.3852289319038391, 0.5731119513511658, -0.12120083719491959, -1.430687427520752, -0.48913073539733887, -1.2820963859558105, 0.17080844938755035, 0.5410427451133728, 2.54046893119812, 0.9155532717704773, -1.0865838527679443, -0.4677869379520416, -0.6407029628753662, 0.2564902901649475, 0.8954421281814575, 0.8711786270141602, 1.8414900302886963, -0.02700069360435009, -0.9770531058311462, -1.8942420482635498, -0.40364620089530945, -2.012385845184326, -0.13707995414733887, 0.4655713438987732, -0.6180886626243591, -0.3987901508808136, -0.7120717167854309, -1.592300295829773, 1.0750670433044434, -0.3331851363182068, 0.321970671415329, -0.8052346706390381, 0.10565979033708572, 0.46758875250816345, 0.3932020962238312, -0.8020198345184326, -0.6038709878921509, 0.43816444277763367, -0.10741584002971649, 0.5151717662811279, 0.005943071097135544, -0.15498098731040955, -1.337766408920288, 0.13773371279239655, 0.8718459010124207, -0.54947429895401, -0.28436189889907837, -0.7025659680366516, 0.9105467200279236, -1.0045589208602905, -0.2845178246498108, 0.4763076901435852, -0.5214029550552368, -0.2920513153076172, -0.24867480993270874, 0.40208008885383606, 0.5820979475975037, -0.8256908059120178, 0.43720099329948425, 0.12869231402873993, 0.8364096879959106, 1.039391279220581, -0.0005697169690392911, -0.07204743474721909, -1.1909613609313965, 0.974860429763794, 0.40722233057022095, -0.8839735984802246, -0.6708478331565857, -1.668073296546936, 1.127326488494873, 0.5720078349113464, -0.19335409998893738, -2.217341661453247, 0.739743709564209, -2.7901291847229004, -0.08319389820098877, 0.8185886144638062, 1.36551833152771, -0.6598122715950012, -0.22658495604991913, 0.6280312538146973, -2.3705742359161377, 0.9307432174682617, 0.05298648029565811, -0.47429919242858887, 1.0843149423599243, 2.024880886077881, 0.3983192443847656, 0.9115400314331055, -0.6012911796569824, -1.308058500289917, -0.024020621553063393, 0.12177184224128723, -0.16132016479969025, 0.5356472730636597, -0.7273973822593689, -1.566377878189087, -1.7304106950759888, -0.800755500793457, -1.3363008499145508, -0.23703446984291077, 1.8428140878677368, 0.30566516518592834, 1.2517290115356445, 1.2999635934829712, 1.9182043075561523, 0.2733851373195648, -1.1498292684555054, -0.24702690541744232, -1.3657585382461548, 2.026146173477173, 0.254818856716156, -1.0909472703933716, -0.33315879106521606, 0.0420292392373085, -0.5333801507949829, -0.33849915862083435, -1.3146384954452515, -0.7070134878158569, 0.7605912685394287, -0.4416685998439789, -0.7243619561195374, -0.19015447795391083, 0.22160935401916504, 0.8658503890037537, -0.6368967890739441, 0.410036563873291, -0.7951487302780151, 0.053584132343530655, -1.189139723777771, 0.40321260690689087, -0.6544017791748047, -2.4992778301239014, 0.15068793296813965, -1.4225419759750366, -0.1616247445344925, -0.3804740905761719, -0.2938954532146454, 1.07016921043396, -0.08667164295911789, -0.7665894627571106, 0.7020094394683838, -0.7531463503837585, 0.7047160267829895, 0.2162618339061737, 0.7531126737594604, -0.2673886716365814, 0.9177689552307129, -0.5120124816894531, 0.07577218115329742, -0.5498369336128235, 1.6073848009109497, 1.459247350692749, -0.4610886871814728, 1.4451924562454224, 0.8090338706970215, -0.9946588277816772, 0.2020435780286789, 0.45158660411834717, -1.3601421117782593, -0.7841973900794983, -0.5121027827262878, -0.22384317219257355, -0.6929153203964233, -0.20226319134235382, -0.9315341114997864, 0.8252267837524414, -0.05742732807993889, 0.3018602132797241, 1.6720716953277588, -0.5569708347320557, -0.1829541176557541, 0.3248327076435089, -1.2323542833328247, -0.5697593688964844, 0.30570095777511597, -0.20674632489681244, -2.9054863452911377, -0.802945613861084, 1.3926780223846436, 0.17959731817245483, 0.2803923487663269, -0.6858307719230652, -0.48949047923088074, 0.22767707705497742, 1.164719581604004, -0.18806496262550354, -1.36738920211792, -0.5737788081169128, -2.6531574726104736, 1.077311635017395, -0.2617221176624298, 0.8617343306541443, 1.026509404182434, 0.02927115373313427, -0.7375163435935974, -1.0866585969924927, 0.2444542944431305, 1.0291080474853516, -0.49199050664901733, -1.053625464439392, 0.4702368676662445, 0.8206045031547546, 1.2352261543273926, -0.6995705366134644, -0.2569165527820587, 0.3255859613418579, 1.248494029045105, 1.6149883270263672, -0.28539109230041504, 0.1421036720275879, -1.268248438835144, -1.9623812437057495, -0.12177220731973648, -0.1999528855085373, -0.7124049663543701, -2.027052164077759, -1.3492333889007568, 0.5624107718467712, 0.04929574951529503, 0.4744005799293518, 0.06090587377548218, -2.124997138977051, -0.5500145554542542, 0.3905300498008728, 0.11244217306375504, -2.031865119934082, -0.6730713844299316, -1.2927627563476562, -0.3051789700984955, 1.436546802520752, -1.4228928089141846, -0.006524009630084038, 0.24342292547225952, 0.6634283661842346, 1.1794531345367432, -0.3102010488510132, -0.6293091773986816, -0.7024713158607483, -0.24013836681842804, -0.6402671933174133, -0.47960707545280457, -0.0953502357006073, -0.4673275947570801, -1.3004359006881714, -0.3271371126174927, -0.792276918888092, 0.30334049463272095, 0.60256028175354, -0.6670520305633545, -0.10596220195293427, -0.3057432174682617, -0.5524060130119324, -0.40599963068962097, 1.748854398727417, 0.5116748213768005, -0.6363455653190613, -0.5518854856491089, 0.15195664763450623, 0.4031428098678589, -0.49584200978279114, -0.5795916318893433, 0.9734021425247192, -1.9631036520004272, 0.7126134037971497, 0.6531457901000977, 0.8778486847877502, -0.1830892115831375, -0.35625705122947693, -1.038421630859375, 1.406407356262207, -0.1820441335439682, 1.448839545249939, 0.7296915650367737, 1.1897696256637573, 0.09225841611623764, -0.21524502336978912, 0.238325297832489, -0.1673124134540558, 0.344779908657074, 0.6405996084213257, -0.1039513573050499, 0.2261570394039154, -0.22434523701667786, 0.41105934977531433, 1.0401535034179688, -1.1993892192840576, 1.5031192302703857, -0.6283331513404846, -1.6248310804367065, 1.4581104516983032, 0.49206772446632385, -0.11229106038808823, 1.061928391456604, 0.47973495721817017, -1.3524776697158813, 0.24141621589660645, 0.3685784339904785, -1.6563622951507568, -0.480330228805542, -0.055952273309230804, -0.4746582806110382, -0.41835999488830566, 0.9546527862548828, 0.6703991889953613, -0.059079475700855255, -0.9162837862968445, -0.02779468707740307, 0.9751407504081726, -0.9012677073478699, -0.5248035788536072, -0.8022116422653198, 0.4044486880302429, -0.4718082547187805, -0.32953494787216187, 0.38263362646102905, -0.6335684061050415, 0.28221797943115234, 0.031694915145635605, 0.2630373537540436, -0.1402972936630249, 0.7592757940292358, -0.7989177107810974, -0.2756690979003906, -1.3498173952102661, 0.28923654556274414, -0.5058306455612183, 0.8259330987930298, 0.8896496891975403, -0.11479675024747849, -0.19617201387882233, 0.16131877899169922, 0.5427254438400269, 0.3965360224246979, 0.15854144096374512, 1.9076532125473022, 0.6455006003379822, 0.037995051592588425, 0.5759568214416504, 1.4351803064346313, 0.36868730187416077, 1.5730847120285034, 0.913807213306427, 0.39197495579719543, 0.716852605342865, 0.049487680196762085, 0.049969907850027084, -0.1365688592195511, 0.6397420763969421, 1.0688483715057373, 0.2792050838470459, 1.2755792140960693, 0.028269458562135696, 1.4462757110595703, -0.7513414621353149, 1.3339202404022217, -1.229537010192871, -1.2033436298370361, 1.8110883235931396, 1.5534037351608276, 2.431614398956299, 0.7051674127578735, -0.036294013261795044, 2.3829143047332764, 0.9999034404754639, 0.2396887242794037, -0.07757745683193207, -0.8106399774551392, 0.7465565800666809, 0.7659049034118652, 1.9445339441299438, 0.3789420425891876, -0.36566582322120667, 1.1514348983764648, 0.08708468824625015, -2.3802645206451416, 0.5748118758201599, -0.7756113409996033, -1.5784786939620972, 0.6104536056518555, -2.693416118621826, -0.03356054052710533, -0.5179102420806885, -1.1711857318878174, 0.5363780856132507, -1.1060599088668823, -0.3315383195877075, 0.973495602607727, 0.9328449964523315, -1.2850918769836426, -0.24404828250408173, 0.39718154072761536, 2.0957083702087402, 1.397240400314331, 1.2377865314483643, 0.2324821799993515, -1.2938554286956787, -1.0487651824951172, -0.3490137457847595, -1.0305427312850952, 0.3567255139350891, -0.11244060844182968, 0.5033266544342041, -0.7215938568115234, 0.10818781703710556, 0.4921167492866516, 0.02055053971707821, 1.5131011009216309, -0.5283249020576477, 0.2975202798843384, -1.3754314184188843, -0.4550018310546875, 0.5597396492958069, -0.05017371475696564, -1.3261727094650269, -0.05815942957997322, 0.3222775161266327, 0.17659199237823486, -0.46761080622673035, 1.0988755226135254, -0.4713423550128937, -0.27785998582839966, -1.3937989473342896, 0.6043027639389038, -3.209378719329834, -0.5934001207351685, 0.5929883718490601, -0.7315131425857544, -0.029487453401088715, -1.9864472150802612, -0.23658387362957, 0.20101185142993927, -1.0954054594039917, 0.5128617882728577, 0.3606146275997162, 0.7514625787734985, 1.0321788787841797, 0.229304701089859, -0.7767358422279358, -0.9209269285202026, 0.20623044669628143, -0.11779119074344635, 0.7216094136238098, -0.9186904430389404, 0.33235964179039, -1.0696277618408203, 0.2762737572193146, -0.06981519609689713, -1.3014447689056396, -1.2156344652175903, -0.0524737611413002, -0.09967637062072754, 1.3884973526000977, 0.4682559072971344, -0.6451370716094971, -0.1676330268383026, -2.392955780029297, 0.6677260398864746, 0.1700417399406433, -0.8424677848815918, -0.7307422757148743, -1.0761101245880127, -0.37715575098991394, -1.0157498121261597, -1.4148879051208496, 0.335737943649292, -0.1400945782661438, 1.3845887184143066, 0.04196573793888092, -1.3971604108810425, -0.3817060589790344, -0.8448947072029114, 0.5554342865943909, -0.8068597316741943, -0.5160515904426575, 0.5942420363426208, -0.7019934058189392, -0.19479438662528992, 1.2522969245910645, 0.36550676822662354, 0.6525781154632568, -0.42692282795906067, -1.657668113708496, 1.7808568477630615, -0.888080894947052, 0.17495238780975342, 0.5978380441665649, 1.376103401184082, -0.6456934809684753, 0.1104326918721199, -2.820871114730835, 0.9977050423622131, 0.21649053692817688, 1.2418760061264038, -1.5065079927444458, 0.54496830701828, 0.8484724760055542, 0.16563807427883148, -0.8904263973236084, -0.8945937156677246, 0.8925453424453735, -0.03946126997470856, -0.15594063699245453, 0.6818313002586365, -1.1253074407577515, 0.9917252659797668, 0.9269959926605225, -0.7641125917434692, 0.6078054308891296, 1.9558939933776855, -0.6176097393035889, 0.49557816982269287, 0.038032665848731995, 0.44993552565574646, -1.929822564125061, 1.26396906375885, 0.6239370703697205, -0.8229089379310608, -0.1114228367805481, -0.760627031326294, -0.83723384141922, -1.821590542793274, 1.3106085062026978, -0.8736895322799683, 0.657602846622467, -0.7133800983428955, 0.5571033954620361, -1.4118187427520752, 1.3450391292572021, -0.3877636790275574, 1.0120482444763184, 1.926449179649353, 0.05594348534941673, 0.3424561619758606, 1.1371456384658813, -2.594846487045288, 0.47096511721611023, 1.3226412534713745, 0.19074001908302307, -1.3008146286010742, 1.0475983619689941, 1.027880072593689, 1.0983953475952148, -0.01587699167430401, -0.020407432690262794, 0.1907886266708374, -0.9004887342453003, -0.44398683309555054, -1.1773582696914673, 2.1389412879943848, -0.9334637522697449, -0.30957862734794617, 0.7317975759506226, -0.14927184581756592, -1.1820110082626343, 0.5906558036804199, -1.1817834377288818, -1.1836633682250977, -0.6648406386375427, 0.010601609013974667, -0.19735011458396912, 0.45339322090148926, -1.442341923713684, -0.18197456002235413, 0.7346146702766418, 0.6690573692321777, -0.8464768528938293, -0.03222019970417023, -2.329785108566284, 0.9363049268722534, 0.8462883234024048, 0.9054509401321411, 2.398632526397705, -0.6374407410621643, 0.29159820079803467, 1.4204318523406982, 1.9840351343154907, -0.3708465099334717, -1.3858014345169067, 0.07632385194301605, 0.8215847015380859, -0.5492836236953735, -0.5631285309791565, -1.2133044004440308, -2.5692427158355713, -0.9961087107658386, 0.51873779296875, 1.136979341506958, 0.4094366729259491, -0.019821831956505775, 1.137013554573059, 1.0844746828079224, 1.4338672161102295, -0.6380432844161987, 0.5604246854782104, -0.29645612835884094, -2.074544668197632, -0.2467430680990219, 0.6534289717674255, 0.33322563767433167, -0.021955592557787895, 0.2339768409729004, -0.4259001910686493, 1.222138524055481, -0.20937959849834442, 0.34858477115631104, 0.5674624443054199, -0.9489673972129822, -0.308504581451416, 0.7678006887435913, -0.21910202503204346, -1.388549566268921, 0.08449771255254745, 0.6186257600784302, -0.4415939152240753, -0.763798177242279, 0.6660840511322021, 0.645968496799469, -1.0936089754104614, -0.22737836837768555, 0.20258130133152008, -1.1871280670166016, 1.1990829706192017, -0.8519478440284729, -1.6557040214538574, 0.05032290145754814, -1.336447834968567, 0.2840350866317749, -0.1599188596010208, 0.9636701941490173, 0.2347317636013031, 0.23230555653572083, 0.6012028455734253, -1.6251304149627686, 0.3976265788078308, -0.8171606659889221, -1.4577218294143677, 0.20062138140201569, -0.44193968176841736, 1.0051064491271973, 0.8509167432785034, 0.5187353491783142, -0.341263085603714, -1.1309880018234253, -1.0618617534637451, -0.6919251084327698, -1.3650791645050049, 0.39175066351890564, -0.8941652774810791, 0.8028221130371094, -0.15034762024879456, 0.7111386060714722, -0.0238630548119545, -0.2490847408771515, -0.42350175976753235, 0.522964358329773, -0.36746856570243835, -0.5163558125495911, 0.13467369973659515, -0.07425010949373245, 1.3540899753570557, 0.10937002301216125, -0.7712393999099731, -1.4748047590255737, -1.0002301931381226, 0.6778748631477356, 0.58184415102005, 0.2803175449371338, -0.2469150424003601, 0.018399665132164955, 0.6774027347564697, 0.9100720882415771, -0.9427134990692139, -0.6632291674613953, -0.2982042133808136, 1.0749975442886353, 1.7834168672561646, -1.408873438835144, 0.31878283619880676, -0.4390455484390259, 1.6977956295013428, -1.0164848566055298, -1.068669319152832, -0.8052303791046143, 0.8590851426124573, 0.1901574730873108, 0.3766143023967743, 0.059489887207746506, 0.9141091108322144, 0.3085559904575348, 1.0986100435256958, 0.500749409198761, 0.16535669565200806, 0.0529317781329155, -1.450943946838379, 2.54099702835083, 1.0763226747512817, 1.0686041116714478, 0.3254357874393463, 0.9530085325241089, 0.32634779810905457, -0.9762446284294128, -0.9001238346099854, 1.0203927755355835, 0.8563060760498047, 1.1207244396209717, 0.7654998898506165, 1.7518867254257202, 0.2245408296585083, 0.5574455857276917, -0.3278140723705292, -2.623904228210449, -1.312856912612915, 0.9926464557647705, 0.30087852478027344, -0.7297325134277344, -1.0761910676956177, 0.0769042819738388, -0.3727877140045166, -1.6156277656555176, -0.8692403435707092, -0.8191598653793335, 0.22691839933395386, 0.9802839756011963, -0.5279707312583923, 0.6186654567718506, 0.33814260363578796, 1.5015641450881958, 1.1618354320526123, 0.493988960981369, 1.3844859600067139, 0.7414339184761047, -0.9080285429954529, -0.6761083602905273, -0.5575858950614929, -0.6833892464637756, -0.8114391565322876, -0.4308142364025116, 0.16121399402618408, 2.500312089920044, 0.8137304186820984, -1.7381296157836914, 0.4026550054550171, 0.8773110508918762, 0.8633256554603577, -0.3513473570346832, -0.9255152344703674, -0.5280545353889465, 0.6892404556274414, -1.4289671182632446, 1.2306684255599976, 1.2771695852279663, 0.24383459985256195, -0.8262377977371216, 1.3158999681472778, 1.0098246335983276, 1.6209319829940796, -1.8020838499069214, -1.7990217208862305, -0.25354039669036865, 0.7428227663040161, -0.8615741729736328, -0.07871001213788986, -0.8631320595741272, -1.0491135120391846, -0.8539043068885803, 0.7936334609985352, -2.545877695083618, -0.5614486932754517, -1.127042531967163, -1.0958688259124756, 0.8951970934867859, -0.6411380171775818, 0.6227720975875854, -0.12756718695163727, -0.34730780124664307, -0.8119890093803406, 0.7318814396858215, 0.23109984397888184, -1.008239507675171, -0.21778623759746552, -0.18192927539348602, -1.575668454170227, 0.4050636291503906, -0.4999028444290161, 0.6457343697547913, 2.4648425579071045, 0.7209489941596985, 0.370286226272583, -1.0983288288116455, -0.5825972557067871, 0.6987891793251038, -1.2121690511703491, -0.7086141109466553, 0.3585400879383087, -0.318498432636261, 1.765479326248169, 0.24055930972099304, 0.880040168762207, 1.0528438091278076, 0.45281144976615906, -1.5997157096862793, 0.4393913447856903, 0.13312552869319916, -0.02470420114696026, 2.3047432899475098, -0.5428563952445984, 1.3802473545074463, 1.1700477600097656, 1.248348355293274, -0.6274263262748718, -0.15346497297286987, -1.2681649923324585, -0.44335028529167175, -0.5095714330673218, -1.212151288986206, -0.014477148652076721, -2.8361783027648926, -0.6363583207130432, 0.4859991669654846, -0.9195978045463562, 0.4081951677799225, 0.6667873859405518, -0.6524428725242615, -1.323925256729126, 1.565028429031372, 0.7911509871482849, 1.139137625694275, 0.30776548385620117, -0.2726023495197296, 1.3058608770370483, -0.17960159480571747, 0.5552458167076111, -0.8575539588928223, -1.3161709308624268, 2.17781138420105, 1.7783703804016113, 0.6721544861793518, 0.030796166509389877, -0.6000433564186096, -0.9844650626182556, 1.2017558813095093, 1.1334325075149536, -0.12421587109565735, -0.08306890726089478, -0.13659417629241943, 1.3033643960952759, -0.1080002412199974, 0.652644693851471, 1.0736525058746338, 0.5761659145355225, 1.0117067098617554, 0.3283841013908386, 0.7249724864959717, -0.19870010018348694, 1.11549711227417, 0.19435414671897888, -0.22441616654396057, 0.4776882529258728, -1.9720072746276855, 2.4553451538085938, -0.02514241822063923, 1.3882980346679688, 1.3058172464370728, -0.6177647113800049, -0.13077539205551147, -0.350880891084671, -0.12191921472549438, -0.746803343296051, 1.8048436641693115, 0.10584747791290283, -1.3810962438583374, -1.683234453201294, -0.6249651312828064, -1.2631133794784546, -0.5209771990776062, -1.4941715002059937, -2.3180854320526123, -0.21091198921203613, 1.2984464168548584, -0.15845593810081482, -0.7919344305992126, 1.0656629800796509, 1.692514419555664, 0.5841474533081055, 0.353731632232666, 0.7110145092010498, 0.787646472454071, -1.5734654664993286, 1.0803853273391724, 0.19035400450229645, 0.13974733650684357, 0.22396442294120789, 1.0789436101913452, -0.03724118694663048, 0.045761436223983765, -0.7526825070381165, -0.8953517079353333, -1.2566903829574585, -0.6798192858695984, 0.5670883059501648, -0.2462388575077057, -1.0422383546829224, -0.13966941833496094, 0.6269130706787109, -0.9202399253845215, 1.4903740882873535, -0.1446840912103653, 0.08942698687314987, -0.7976913452148438, 0.5446481704711914, 1.5146359205245972, 0.9469380378723145, -1.3689312934875488, -1.5915430784225464, 1.1923984289169312, 0.31835293769836426, 0.5067513585090637, 0.48560982942581177, 0.35642209649086, -1.075945496559143, -0.8408649563789368, 0.4995635449886322, 0.8887091279029846, -0.6182632446289062, -1.9978015422821045, -1.1590615510940552, -0.9145305156707764, -1.8170164823532104, 0.6367102861404419, 0.035068780183792114, -0.10055415332317352, -2.1059601306915283, -1.4279011487960815, -0.06358896940946579, -2.4663078784942627, 0.10522940754890442, -1.4170212745666504, 0.9700515866279602, -0.47737324237823486, 0.11696738749742508, 1.2018096446990967, -0.10718511044979095, 1.258530855178833, -2.244468927383423, -1.3484375476837158, -0.8937314748764038, 0.6158507466316223, -0.15433885157108307, -0.7967312335968018, 0.11737757176160812, 0.8670182228088379, -0.6396586894989014, 0.3028016686439514, 2.0224874019622803, 1.6838505268096924, -0.8548743724822998, -0.10980895161628723, -1.173912763595581, -0.41244328022003174, -0.6091004014015198, 0.1410447210073471, -0.9808842539787292, 0.411827027797699, 0.0429324209690094, -0.6521705389022827, -0.2621729373931885, 1.0932414531707764, 0.28711044788360596, 2.0159120559692383, -0.276947021484375, 0.22834599018096924, -1.8943508863449097, -0.8818051815032959, -1.798744559288025, 0.7977569699287415, 0.19276611506938934, 0.2929600477218628, -0.36143264174461365, -1.1020194292068481, 1.0353387594223022, 0.5171114206314087, 1.4742223024368286, -0.4075736105442047, -0.1778571456670761, 1.3044790029525757, -0.027664532884955406, -1.6769853830337524, 0.5804199576377869, -0.04482405260205269, -1.4917787313461304, -0.7341581583023071, 1.218570590019226, 1.0649269819259644, -1.0451511144638062, 1.098983645439148, -0.6032511591911316, -0.5070504546165466, -1.3645148277282715, -1.0753618478775024, 0.046602196991443634, 0.39430370926856995, -0.7248289585113525, 3.834709405899048, -0.9858259558677673, 0.542789876461029, 0.5393344759941101, -0.7688992023468018, 1.114060401916504, 1.5352195501327515, -1.5357199907302856, -0.15938423573970795, -0.7626888155937195, -0.4300991892814636, -1.3645012378692627, 0.9217755794525146, 1.4290945529937744, -0.20049406588077545, 1.0743980407714844, 0.8239554762840271, 1.1829880475997925, 0.1098078116774559, 0.21925070881843567, -2.609938144683838, 1.654982089996338, 0.13316801190376282, 0.2808727025985718, -0.5038378238677979, -0.9476848244667053, -0.0031136993784457445, 0.8436757922172546, -1.178061842918396, 0.09740804880857468, 0.37286555767059326, -0.8970929980278015, -0.6932978630065918, -1.1137306690216064, -0.7680066823959351, -0.029986314475536346, -0.017853721976280212, 0.6908767223358154, 0.057189133018255234, -0.9167511463165283, -1.0385388135910034, -1.372725486755371, 0.3731096088886261, 0.33792194724082947, 0.22888539731502533, -0.9955950379371643, -2.589114189147949, 0.26132306456565857, 0.5606557726860046, 1.1390219926834106, -1.7493526935577393, -0.17776410281658173, -0.2922894060611725, -0.7887979745864868, -0.28631946444511414, 0.9536950588226318, -1.4182226657867432, 1.284277319908142, -2.0727615356445312, -0.30984213948249817, 0.3690531551837921, -0.6911920309066772, -0.834821343421936, -0.7494384050369263, 0.13555093109607697, -2.697082281112671, 1.094892978668213, 0.32131439447402954, -0.11172253638505936, 1.06027352809906, -0.7938839197158813, -1.6738457679748535, 1.37330162525177, 1.2889443635940552, 1.3192040920257568, 0.45521363615989685, -1.0215301513671875, 0.47532302141189575, -0.6203112602233887, 1.0374575853347778, -0.2631681263446808, 0.6915573477745056, 0.8986833691596985, -0.980522632598877, -0.2582075297832489, 2.050370216369629, -0.6830343008041382, -0.268978089094162, -1.2795720100402832, 0.7416931986808777, 0.09560608118772507, -1.7249423265457153, 0.009572348557412624, -0.5936405658721924, 1.957699179649353, -1.332959771156311, 1.182310938835144, 0.3277136981487274, -1.368567705154419, 0.7854982614517212, 0.9179277420043945, -0.8571447730064392, 0.14816385507583618, -1.4039124250411987, 0.2551252245903015, 1.1899806261062622, -1.2199172973632812, 0.5189706087112427, 2.012892246246338, 1.2487159967422485, -0.38682684302330017, 1.5121983289718628, 0.7249534726142883, -0.4690599739551544, 1.5344525575637817, -0.5688627362251282, -0.5198912024497986, -0.8895718455314636, -1.3140827417373657, -0.10914167761802673, 0.9830882549285889, 1.6212469339370728, 0.22757352888584137, 2.086901903152466, -1.2782400846481323, -2.164076566696167, -0.3624712824821472, -1.5976440906524658, -0.37105897068977356, 2.2746944427490234, 1.1458786725997925, 0.759768009185791, 0.6442384123802185, 1.0783294439315796, -0.5760116577148438, 0.5331632494926453, 1.7773138284683228, -1.7768226861953735, -0.18135353922843933, 0.4663136899471283, 0.14879761636257172, -0.45576027035713196, 1.6039879322052002, 1.4789763689041138, 0.5152400732040405, 1.0887936353683472, -1.3718992471694946, 0.9099281430244446, -0.13597959280014038, 0.017241686582565308, -0.45127072930336, 0.18127691745758057, 1.667165756225586, 0.6355376839637756, -0.8309069871902466, 0.4974353313446045, -0.9144512414932251, 1.2754796743392944, 1.2819782495498657, -0.4218807518482208, -0.7251208424568176, 1.5121687650680542, -0.39578738808631897, 1.4320465326309204, -0.9109520316123962, -0.9571589231491089, -1.3033818006515503, -0.5251776576042175, -1.3051332235336304, 0.32852065563201904, 1.5404032468795776, -0.014961445704102516, 0.7962262034416199, -0.46697285771369934, 0.27965492010116577, -0.25994738936424255, -0.6334831714630127, -2.2352709770202637, -0.36220207810401917, -1.29300057888031, 2.2680790424346924, 0.5390358567237854, -0.6873663067817688, 0.1522872895002365, -0.6129352450370789, 1.7349156141281128, -0.9139836430549622, -0.17778795957565308, 0.9262533187866211, -1.321785807609558, 0.7355216145515442, 1.1949489116668701, -0.34600627422332764, -0.5774654150009155, 0.6694685816764832, 1.093686580657959, -0.2534004747867584, 2.6135871410369873, -2.2660892009735107, -0.18401126563549042, 2.106370687484741, -0.607728123664856, -1.4289449453353882, 0.5782545208930969, 0.909423828125, 0.3814520239830017, -1.9141796827316284, 0.9436922073364258, -1.3415591716766357, 1.7187870740890503, 1.0944300889968872, 0.1831832081079483, -0.4318731427192688, 0.4818909168243408, 0.6343533992767334, 0.2114648073911667, -2.2520029544830322, -0.4497547149658203, -1.1077536344528198, -1.3705722093582153, 1.4113521575927734, 0.5353280901908875, -0.5332382917404175, -0.8715000748634338, -1.3593426942825317, -1.4515252113342285, -0.665993332862854, -0.10510946065187454, 0.9497950077056885, -1.0561480522155762, -1.0405588150024414, -2.358346939086914, 0.6008450984954834, -0.07191658020019531, -1.6212562322616577, 1.0402305126190186, 0.3072870373725891, 0.41658148169517517, -0.6232201457023621, -0.6002256274223328, 0.07816474139690399, 0.09976093471050262, -0.4200925827026367, -0.9739251136779785, -0.8482878804206848, -0.9935646057128906, 1.4858981370925903, -0.19688989222049713, 2.1767923831939697, -0.8895025849342346, 0.6742618083953857, -0.2880052328109741, 0.8613616824150085, -0.4593504071235657, -0.42897912859916687, -1.1425143480300903, 0.2048081010580063, -0.07515152543783188, -0.15051592886447906, -1.0503573417663574, 1.1478095054626465, 2.8864026069641113, -1.201786994934082, 2.2035889625549316, -0.6527217626571655, 0.5222858190536499, -0.8978546261787415, -0.45122256875038147, -1.0221221446990967, 0.6794273257255554, -1.703348994255066, 0.8508414030075073, 1.2062962055206299, 0.3693041503429413, 0.7186859846115112, -3.3260364532470703, 0.007169095333665609, 0.24401354789733887, -0.6117013692855835, 0.24175122380256653, 0.07191399484872818, -1.1012850999832153, -2.4467923641204834, -2.1947214603424072, -0.8127919435501099, 0.2681616246700287, 0.08629845827817917, 0.3335411250591278, -0.634634792804718, -1.9146567583084106, -0.17733295261859894, -0.4311358630657196, 0.9075034856796265, -1.6586753129959106, 0.5920937657356262, 0.8472951054573059, -0.21468907594680786, -0.6348043084144592, -0.21088166534900665, -0.6886764168739319, 0.23632560670375824, -1.108597755432129, -0.885799765586853, 0.4448966085910797, 0.2867518663406372, -2.516939878463745, 0.07880334556102753, -0.5373640656471252, 0.2547571361064911, -0.2639386057853699, 0.012639559805393219, -0.3090626299381256, -2.251589059829712, -0.21164833009243011, -1.396329641342163, 0.33851221203804016, -0.8166630268096924, 1.5665639638900757, 1.5496193170547485, 0.06592021882534027, -0.8020036816596985, 0.5747286081314087, -1.2154940366744995, -0.16244620084762573, 0.5649512410163879, 0.8747410774230957, 1.5693069696426392, -0.7099413275718689, 1.6926881074905396, 0.6111883521080017, 0.732969343662262, 0.3767209053039551, 0.5561541318893433, -0.7062957286834717, 0.5013231635093689, 1.8081756830215454, -1.0010367631912231, -1.6678909063339233, 1.8272639513015747, -1.4905370473861694, -0.19357995688915253, 1.0336955785751343, 0.3304646611213684, -0.873461902141571, -1.2943543195724487, 0.6752851605415344, -0.7589446902275085, -0.15795831382274628, 1.1293176412582397, 0.4042206406593323, 2.2961785793304443, -1.3956685066223145, -0.4141097664833069, -1.1769410371780396, -0.43802329897880554, -0.4416065216064453, -0.2571031153202057, -1.1788313388824463, 0.8952701687812805, 2.7561943531036377, -0.2330397069454193, -2.3346855640411377, -0.8095805644989014, -0.08860665559768677, 1.0053906440734863, 0.3477693796157837, 1.3423784971237183, -0.5898228883743286, -0.4280944764614105, 0.6908577680587769, -0.39058974385261536, -2.6301209926605225, 0.27958935499191284, -0.5384750366210938, -0.2568994164466858, -0.9494255185127258, 2.338550567626953, -0.06189917400479317, -2.245204448699951, -0.3140415847301483, 0.8033521175384521, -1.457506775856018, -0.17263922095298767, 0.7245713472366333, 0.3358543813228607, -1.6599256992340088, -0.0006714219343848526, -1.5536361932754517, 0.23774050176143646, 0.8777776956558228, 1.1568058729171753, 1.5583845376968384, 0.397182822227478, -1.448314905166626, -1.1036336421966553, -0.5575016140937805, -1.0577083826065063, -0.8643127679824829, 1.644889235496521, 0.16615255177021027, -1.518456220626831, -1.3408117294311523, 0.44026848673820496, -0.46366676688194275, 0.9307281374931335, 0.9677506685256958, -2.1294472217559814, 0.47080880403518677, -0.7396473288536072, -1.217282772064209, 0.0685470849275589, -0.7917225360870361, 0.7058118581771851, -1.3268680572509766, -0.7877678275108337, -0.042827997356653214, -1.3439558744430542, -0.577728807926178, -0.7942899465560913, -0.11943810433149338, -1.4137821197509766, 1.8623489141464233, 1.4991616010665894, 0.6208921074867249, -0.7749141454696655, -0.21617674827575684, 1.0534980297088623, 0.25322332978248596, -0.008649824187159538, 0.12451756000518799, -1.6509525775909424, -0.26271867752075195, 2.1063003540039062, -0.32871055603027344, -0.19641916453838348, -1.0590448379516602, 0.8065124750137329, 0.8718850016593933, -0.15728910267353058, 1.5767693519592285, 1.2918963432312012, 1.098570704460144, -0.6861182451248169, -0.7643784880638123, -0.06670809537172318, -0.8627520799636841, -0.2754586935043335, 0.35562416911125183, -1.4474235773086548, -0.36417049169540405, 0.26313817501068115, 0.6502074003219604, 2.6159772872924805, -0.04307728260755539, 0.8638482689857483, 0.07793249189853668, 1.0568733215332031, -1.3844548463821411, -2.7835373878479004, -1.150050163269043, -1.0727976560592651, 1.8723171949386597, -0.07355774939060211, 0.7854729890823364, 1.590017318725586, -0.18570296466350555, 0.7129083871841431, -1.4661165475845337, 0.4202885627746582, -1.1480967998504639, -1.7230271100997925, 0.9892264604568481, 0.8186166882514954, -0.19061322510242462, 1.555416464805603, -0.20795640349388123, 0.8404645323753357, -1.5352755784988403, 0.08548379689455032, -2.8424055576324463, -0.40439528226852417, 0.28942790627479553, -0.4798680245876312, -1.176928162574768, 0.7447832226753235, -0.32772910594940186, 0.9783983826637268, -0.3719603717327118, -0.09198091924190521, -1.7942676544189453, -1.2616121768951416, -0.778520405292511, 1.923865556716919, -0.6638283133506775, -0.23246634006500244, -1.8490968942642212, -1.9338124990463257, 0.26553595066070557, -0.122193843126297, 0.6853318214416504, 2.1913692951202393, -1.1448943614959717, -1.0137465000152588, 0.97347491979599, -1.7658401727676392, -1.6313185691833496, -0.3412921726703644, -0.30187681317329407, 0.19343964755535126, 0.20523642003536224, -0.5631827712059021, -1.323896884918213, -2.2921063899993896, 0.06635086238384247, -0.18891355395317078, -2.6872670650482178, 0.4516340494155884, -0.9491585493087769, -0.6822054386138916, -0.05018240585923195, -0.7558555603027344, 0.7999964952468872, 0.27295824885368347, 0.653313398361206, 1.9157986640930176, 0.3451041281223297, -0.6419153213500977, 0.8628770709037781, -0.8309107422828674, -2.952505350112915, 0.743628978729248, 0.5597797632217407, -1.1991633176803589, 0.9401987195014954, 1.8480536937713623, 0.5065088272094727, -0.06284290552139282, 0.08514559268951416, -0.7518012523651123, 0.4672945439815521, 0.6441845893859863, 1.0989903211593628, 0.008431484922766685, 0.8466165661811829, -0.9723979830741882, -0.6071648001670837, -0.2841186225414276, 0.9098027348518372, 1.6719996929168701, 0.7992660999298096, -1.1280733346939087, 0.028437698259949684, 0.87940913438797, -0.3313637971878052, 1.8119568824768066, -1.178390383720398, -1.2067090272903442, -0.1197362169623375, -1.2500354051589966, 0.5811348557472229, -1.7090202569961548, 1.491383671760559, -0.020450761541724205, 1.0759092569351196, -0.5151193141937256, 0.015834404155611992, -2.3458304405212402, -2.029792547225952, -0.3623765707015991, -1.0806233882904053, -1.706259846687317, -0.730236291885376, -0.4190022051334381, 0.593527615070343, 0.778338611125946, -1.746010661125183, 0.2505556643009186, 0.15614783763885498, -0.6043074131011963, -0.4145328998565674, 0.369329571723938, -1.4833130836486816, -0.31609880924224854, 0.8106671571731567, 2.301679849624634, 1.29212486743927, 0.7682818174362183, -2.1589035987854004, -0.26620516180992126, -1.4523905515670776, 2.251237630844116, 0.1222912073135376, -1.4079084396362305, -1.936023473739624, 0.01116381399333477, -0.37986549735069275, -1.5331544876098633, 1.1958409547805786, 0.3500571846961975, -0.9459115266799927, -0.011350017972290516, 2.4273860454559326, -1.0191811323165894, -0.4487311840057373, -1.8085335493087769, -0.9449382424354553, -1.2726401090621948, 1.3656713962554932, -0.6672436594963074, 0.4198586046695709, 0.4624183773994446, -0.6358945369720459, 0.9011989235877991, -0.08176244050264359, -0.8658562302589417, 0.2151746153831482, -1.3182868957519531, -0.3868866264820099, -0.19346728920936584, 0.6175569891929626, 0.11683794856071472, -1.164637804031372, -0.6253565549850464, 0.1346852034330368, -0.28452250361442566, -0.004064058884978294, 0.3994336724281311, -1.7521823644638062, 0.3644990921020508, -0.6673902273178101, -0.2648918330669403, -0.49647045135498047, -0.5876618027687073, 0.04881511628627777, -0.6454327702522278, -1.6008228063583374, -0.28256985545158386, 0.031109681352972984, -0.6401470303535461, 0.016470789909362793, -1.1760700941085815, -0.8955557346343994, 1.8050585985183716, -0.25691869854927063, -1.154992938041687, -0.5014397501945496, -0.26304060220718384, -0.41035398840904236, -1.5447766780853271, 0.13257867097854614, 0.6291429400444031, 0.35251954197883606, -1.9348700046539307, -0.3282528519630432, -0.787696361541748, -0.01255748514086008, -0.14932195842266083, 0.28559839725494385, -0.008690758608281612, -0.8166991472244263, -0.547045111656189, -0.24535585939884186, 0.4845406711101532, -2.300961494445801, -0.14831124246120453, -1.5604122877120972, -1.4154435396194458, -0.7752512693405151, 0.3299091160297394, 0.6109015345573425, -1.9295389652252197, -0.5380843281745911, 0.9280198216438293, -0.39571136236190796, -1.0663379430770874, 1.1434712409973145, 1.2170308828353882, 0.36001190543174744, 0.8064438700675964, -0.0782892033457756, 2.5445549488067627, -0.6723807454109192, -0.36835357546806335, -1.410308837890625, 1.5448213815689087, 0.18158748745918274, 0.30032360553741455, -0.7270719408988953, -1.1972547769546509, -0.3268533945083618, 2.0976247787475586, 0.006329705473035574, -0.13420113921165466, -0.0007309917127713561, -0.14267900586128235, 0.9659155011177063, -0.4644286036491394, 0.08388526737689972, -1.5794588327407837, -0.5459269285202026, 0.40912145376205444, 0.8714861273765564, -1.7548578977584839, 2.0474467277526855, 0.5138066411018372, 0.7879856824874878, 0.02160036377608776, 0.6193763613700867, 0.5391960144042969, -0.6544656157493591, 0.5632607340812683, 0.001445890637114644, -2.083613872528076, -1.028296947479248, -0.2568303048610687, -1.2004544734954834, 0.2018868327140808, 0.3987463116645813, -0.34224623441696167, -0.1071120873093605, 0.9142850041389465, 2.6367108821868896, 0.06870350241661072, 0.1307476907968521, -0.7648055553436279, -1.1062477827072144, -0.7477232217788696, -0.09793435782194138, 0.5373862385749817, -0.9407786130905151, -0.1039859876036644, 1.2010642290115356, 1.2472718954086304, 0.1337042897939682, 1.1570278406143188, -1.101539969444275, -1.952638864517212, -0.45787885785102844, 1.441542625427246, 0.5772119164466858, -0.09982946515083313, -0.9268798828125, -0.8961565494537354, 0.09995225816965103, 2.3089442253112793, -1.0239150524139404, -0.4789804518222809, -1.0639129877090454, 0.07821953296661377, 1.3454352617263794, -1.087247371673584, -0.22813822329044342, -0.024342060089111328, 0.6407760381698608, 0.2667991518974304, 1.412157654762268, 1.2164477109909058, -0.29271838068962097, 0.41688352823257446, 0.2303149402141571, 0.6119796633720398, 0.014065033756196499, 1.2781703472137451, -0.17438212037086487, 1.4134769439697266, -1.6052865982055664, -0.18825815618038177, -1.7642240524291992, -0.8360713124275208, -0.8919859528541565, -0.368056982755661, -0.29593920707702637, 0.3734351694583893, -0.1896502822637558, -1.8159507513046265, 0.11900458484888077, 1.337834358215332, -0.2241055816411972, -0.6843923330307007, 0.34958359599113464, 0.20276246964931488, -1.3487051725387573, 0.41495388746261597, 0.7395660877227783, -0.8723563551902771, 0.21382759511470795, 1.036281704902649, 3.084395408630371, -0.48927584290504456, 0.36994871497154236, 1.2699095010757446, -0.050570741295814514, 0.6302018761634827, 1.3553203344345093, 1.0943259000778198, 1.7241389751434326, 0.404954731464386, -0.3893994688987732, -0.25643959641456604, -0.8543850779533386, 1.0343554019927979, -0.5097631812095642, -0.06403303891420364, -0.2185784876346588, 0.0026065742131322622, -0.07204735279083252, 1.3298838138580322, 0.8530899882316589, 0.336448073387146, -0.7509865760803223, 1.4195903539657593, 1.0644642114639282, 0.8708493709564209, 0.027805963531136513, 0.8062717914581299, 0.8696243762969971, 0.008609460666775703, 0.03591523692011833, 0.9963822960853577, -1.3548702001571655, -0.4156954288482666, 0.3589695394039154, -0.6818220615386963, 0.5988208651542664, 0.8935259580612183, 1.230299472808838, -0.6099364757537842, 1.5193233489990234, 1.01570725440979, -2.4988973140716553, -1.0596009492874146, -0.6073216795921326, 0.40520718693733215, -1.74008047580719, -0.07150368392467499, -1.8542605638504028, 0.20558607578277588, 0.3343481421470642, -1.7654237747192383, 0.1872786283493042, 1.7070183753967285, -1.3198822736740112, -1.9796178340911865, -0.9448503255844116, 0.8661790490150452, -0.29922181367874146, -0.7040921449661255, 0.381064772605896, -0.6562834978103638, 0.8083892464637756, 1.597675085067749, -1.6340149641036987, 0.028091885149478912, 2.8332760334014893, -0.9500271081924438, -0.36065953969955444, 0.7158632874488831, -0.4982813894748688, -1.518784523010254, 1.4600685834884644, 0.29741808772087097, 1.2356420755386353, -0.5286920666694641, 0.233856201171875, -0.007320417556911707, 0.5276665091514587, -0.8066348433494568, 1.5364614725112915, 0.4316859543323517, -0.5307353734970093, -0.6020424365997314, 1.3662378787994385, -1.1486170291900635, 0.3687463104724884, 1.2434006929397583, 0.4240083396434784, -0.7102631330490112, -2.773815393447876, -0.14720158278942108, -0.41039973497390747, 0.2318364381790161, 0.0965098962187767, -1.766619324684143, -1.3340519666671753, 0.04057525843381882, 1.9349159002304077, -0.257342129945755, -0.6494002342224121, 0.08760391175746918, 0.7413060069084167, 1.4901659488677979, -0.4326852560043335, -1.3658852577209473, -2.5162408351898193, -0.3492092788219452, 0.8017280101776123, 1.7621904611587524, 0.09186327457427979, -1.4438514709472656, -1.183129906654358, 0.2774219810962677, 1.0835161209106445, -0.3450509011745453, 0.4811199903488159, 0.6530094742774963, -1.00680673122406, 0.2723953127861023, 0.10078594833612442, 1.562727689743042, -0.31979578733444214, -0.6523028016090393, 0.5751008987426758, 0.9715662002563477, -0.4082637131214142, -0.5792626142501831, 0.25622400641441345, 0.9891210198402405, -1.2344733476638794, -0.28496843576431274, -0.47706353664398193, 0.3467652499675751, 0.9645105004310608, 0.24819058179855347, 0.16673213243484497, 0.04817589372396469, -0.0025513279251754284, 0.480223149061203, 0.4483487904071808, -1.4771367311477661, -0.6624323129653931, 1.6078143119812012, -0.5873028039932251, 0.8377411365509033, 0.6227692365646362, -0.40732628107070923, 1.2233630418777466, -0.6106622219085693, 0.8627969026565552, 1.315779447555542, -0.07264116406440735, 1.2279025316238403, 2.2916862964630127, -0.7599590420722961, 0.4906027019023895, 0.6119392514228821, 1.8291776180267334, -0.9470319151878357, -0.6685928702354431, -0.4326358735561371, 1.0512678623199463, -0.45336854457855225, -0.20111359655857086, -1.1128531694412231, -0.009308888576924801, -0.3093290328979492, -0.40742576122283936, -2.0281245708465576, -0.758183479309082, 2.0025041103363037, 0.4591902494430542, 0.7124625444412231, -1.8303475379943848, 0.2071210891008377, -0.46823444962501526, -0.18080587685108185, 0.8534212112426758, -0.4795382022857666, 0.8187390565872192, 1.0514572858810425, 1.3704447746276855, 0.5032828450202942, 0.586496889591217, 1.8772507905960083, -1.064592957496643, -0.49737274646759033, -0.1500888466835022, -0.23993712663650513, -0.26663148403167725, -0.9856601357460022, 0.8226187229156494, 0.3715756833553314, 0.2590772211551666, -0.9096136689186096, -0.5059481859207153, -1.1735484600067139, 0.028034456074237823, -0.843658447265625, 1.4502497911453247, 1.7037373781204224, -0.35376641154289246, 0.4556569457054138, 0.45134055614471436, 0.08769257366657257, -0.04416084662079811, -0.5715113878250122, 1.0681833028793335, -0.5560846328735352, -0.12411372363567352, 2.362788200378418, 1.5315301418304443, 0.33687877655029297, 0.08427538722753525, -1.1979974508285522, -1.3008136749267578, -1.1420258283615112, 1.1674141883850098, 0.2865769565105438, 0.39112550020217896, 1.0223547220230103, 1.6470016241073608, 0.29188698530197144, -1.0304497480392456, 0.25972694158554077, -0.9000498056411743, -0.3922153115272522, -1.225899338722229, -0.99951171875, -1.6134730577468872, -0.79039466381073, -1.5415581464767456, 1.5682940483093262, -0.20411115884780884, -0.2988348603248596, -0.30062952637672424, 0.6688100099563599, 1.0260530710220337, 1.9279389381408691, -1.2264049053192139, 1.8299813270568848], [1.637292504310608, -0.4792015254497528, 1.764549970626831, -0.10648828744888306, -0.9576772451400757, -0.22312620282173157, 0.25211355090141296, -0.13053834438323975, -3.4961180686950684, 0.09034628421068192, 0.25338080525398254, 0.9648486971855164, -0.13290804624557495, -1.1056056022644043, 0.5707619190216064, -0.7338029146194458, 0.5850000977516174, -1.5175023078918457, -0.7413814663887024, 0.007617324125021696, 1.483099341392517, 0.19024604558944702, -0.33118465542793274, -0.9630721807479858, -0.011646131053566933, 0.21417906880378723, 0.8241207599639893, -1.0340644121170044, 0.6861026287078857, -0.620389997959137, -0.8366904854774475, 0.7678509950637817, 0.4559040665626526, 1.969931960105896, 0.21544693410396576, 0.23448476195335388, -2.292308807373047, -0.36591699719429016, 0.24273885786533356, -1.1313605308532715, 0.45030075311660767, 0.3128611147403717, 0.5605894923210144, -0.8247496485710144, 0.27846863865852356, -0.7019705772399902, 1.498274326324463, -0.9310542941093445, 1.7150510549545288, 0.24836000800132751, -0.5277841091156006, 2.2998621463775635, -0.7460544109344482, 1.2749145030975342, -0.44219350814819336, 1.4051446914672852, -2.223566770553589, -0.22295241057872772, 0.6596823334693909, 0.7000753879547119, -0.7467213273048401, 0.33603647351264954, 0.01918916590511799, -0.5252830982208252, -0.13616614043712616, -1.3211627006530762, -0.2552162706851959, 0.8124535083770752, -0.11871378868818283, -0.6116759181022644, -0.9220231771469116, -0.3438586890697479, 0.7476108074188232, 0.963794469833374, -0.7905071973800659, -1.4316697120666504, 0.1949770301580429, -1.2025229930877686, 0.6345289945602417, -1.000612497329712, -0.3890002369880676, -0.7051914930343628, 0.06828727573156357, -0.655022144317627, 0.7455441951751709, -0.7570385932922363, -0.47421786189079285, -1.8158884048461914, 0.40387845039367676, -0.13544882833957672, -0.5556949973106384, -0.33450379967689514, 2.0114331245422363, -0.7700968980789185, -1.9482303857803345, -0.5113726854324341, 0.2749602794647217, 0.4505157470703125, -0.9987298250198364, -1.6786198616027832, -2.1597299575805664, 0.9753240942955017, -0.16872599720954895, 0.08429557085037231, -1.3706406354904175, -0.06629540771245956, -0.2638178765773773, 1.657920002937317, -0.707231879234314, 0.3017042875289917, -0.46141183376312256, 0.396696537733078, -0.10312367975711823, -1.3067952394485474, -1.5160112380981445, -0.8241890668869019, 0.6754586696624756, 1.1047086715698242, 0.069020576775074, 0.03193291649222374, -0.7863552570343018, 0.032553162425756454, 0.705563485622406, 0.11077616363763809, -0.7385977506637573, 0.27858489751815796, 0.09416945278644562, 1.014941930770874, 1.0567982196807861, 0.15758194029331207, -1.1250869035720825, 0.6642350554466248, -0.5399149060249329, 1.7379857301712036, 0.9880247116088867, -0.4856383204460144, 0.28662192821502686, 0.2638739049434662, -1.7681982517242432, -0.7585340738296509, 1.024706244468689, -0.02436871826648712, 1.130403757095337, 0.3740691840648651, 0.5727119445800781, 1.1032317876815796, 0.6015433073043823, 2.13999342918396, -0.1591910719871521, -1.3791865110397339, -0.26341012120246887, -0.8018876910209656, 0.7360355257987976, 1.1575833559036255, 0.5146268010139465, 1.100726842880249, 1.0287222862243652, -0.09670950472354889, -0.35851138830184937, -1.1078001260757446, 0.4590919315814972, -1.1751103401184082, -0.03167320415377617, -0.4042665362358093, 0.671985924243927, 1.2714760303497314, -1.3020820617675781, 0.584262490272522, 0.16638925671577454, -1.742485523223877, -0.19435696303844452, -0.6597372889518738, -2.0977180004119873, 3.2799530029296875, 1.3314682245254517, -0.6192619800567627, -0.01116064190864563, -0.4312083423137665, -0.5855751037597656, -0.9697437286376953, -1.19912588596344, -0.3863750398159027, -0.9214804768562317, 1.63375985622406, -0.12855760753154755, 0.7827033996582031, 0.7803831100463867, 1.0162568092346191, 1.4562524557113647, 1.6086742877960205, 1.0020567178726196, 1.2347056865692139, -0.25842487812042236, 0.418366014957428, -0.5888444781303406, 0.250145822763443, 1.6561644077301025, -0.034493882209062576, -1.3309400081634521, 1.9476137161254883, 0.6661884784698486, -1.2896257638931274, -1.5442686080932617, -0.02550838328897953, -1.0168553590774536, 0.904371440410614, 0.5723461508750916, -0.18458135426044464, 2.1410398483276367, -2.196938991546631, 0.38036417961120605, 0.51963210105896, -0.24530364573001862, 0.6820755004882812, -0.379965603351593, 0.052752211689949036, -0.6349194645881653, -0.7656412124633789, -0.2509543299674988, -0.16121579706668854, -0.8807930946350098, 1.5039021968841553, 0.003640195121988654, -0.1771591752767563, 0.5199708342552185, -0.49728745222091675, -0.49720293283462524, -0.4878197908401489, -0.7848220467567444, 0.3927675187587738, 0.2175191044807434, -0.1457996368408203, 1.4016244411468506, -0.04367798566818237, 1.1937898397445679, -1.2264126539230347, 1.3307549953460693, -0.9108631014823914, 0.3549957573413849, 0.30461645126342773, 0.11896292865276337, 0.03118441067636013, -0.7641284465789795, 1.6899445056915283, 0.6021515727043152, -2.1842761039733887, -0.17083902657032013, 1.3228679895401, -1.8910472393035889, -0.9703307747840881, 0.36762410402297974, 0.15772108733654022, -1.4604277610778809, -0.8255196809768677, 0.2334369719028473, 1.032834529876709, -1.036220908164978, 2.5303523540496826, 0.7514961957931519, -0.552912175655365, -0.15610460937023163, 0.12180637568235397, -0.1246480718255043, 0.20412340760231018, 0.40811461210250854, -0.47322195768356323, 0.9871371984481812, -0.40133926272392273, -0.3276851773262024, 0.7325190305709839, 0.6866251230239868, 0.472901850938797, 2.29874324798584, 0.7828795313835144, 0.8856074810028076, -0.2743115723133087, -1.8301875591278076, -0.6496202349662781, 0.8752301335334778, -0.7278116941452026, 0.7579255104064941, -0.5672292709350586, -0.40647682547569275, 0.5703198313713074, -0.32297685742378235, 1.8655343055725098, -0.5634534358978271, -0.6885448694229126, -0.8699434995651245, -0.296613872051239, -0.19995218515396118, 0.039637863636016846, -0.15034212172031403, -1.1109888553619385, 0.9999961256980896, 1.3058726787567139, -0.6681363582611084, 0.8606853485107422, 0.6150282025337219, 1.3436007499694824, 0.7118141055107117, 0.6059893369674683, -0.6238356232643127, 0.3587976098060608, -2.609452486038208, 2.588160991668701, 1.4721227884292603, -0.7814859747886658, -1.9075963497161865, -1.7179597616195679, 0.003250989830121398, -1.3066450357437134, -1.21177077293396, -2.2271981239318848, 0.5824468731880188, 0.04404141753911972, 0.5354565978050232, -1.6014357805252075, -1.1044453382492065, 0.008508528582751751, -1.5034849643707275, 0.3734458386898041, -0.3937055766582489, -0.6491544842720032, -1.2107723951339722, 1.036271095275879, 1.3611739873886108, 0.8538968563079834, 1.1734046936035156, -1.2968089580535889, 2.2330586910247803, 2.235896587371826, 1.2626245021820068, -0.6197237372398376, 1.778222918510437, -0.6127306818962097, -0.31611302495002747, 0.6504807472229004, -1.62312912940979, 2.1112823486328125, -0.6018673181533813, -0.390838623046875, -0.3439733386039734, 1.1645761728286743, 2.9415366649627686, 0.9211835861206055, 0.08684375137090683, 1.3875186443328857, -0.3073860704898834, -0.5058428049087524, -1.537175178527832, -0.41631442308425903, -0.8228158950805664, -1.8143380880355835, -0.30925416946411133, -0.9481159448623657, -0.4182226359844208, 0.2749926745891571, -1.0045015811920166, 1.2376819849014282, 0.19749969244003296, 0.49969199299812317, 0.957772433757782, -0.8815332651138306, -1.821006178855896, -0.8046213388442993, 2.0704479217529297, -0.9250879287719727, 0.2167843133211136, -0.38372042775154114, -0.5386956930160522, 0.2362632155418396, -1.3991624116897583, 1.0966848134994507, -0.14702874422073364, -0.8747827410697937, -0.4001745283603668, -1.041670322418213, -0.550404965877533, 0.07800724357366562, -1.3211668729782104, -1.3045393228530884, -0.5205444097518921, 2.1560328006744385, 2.1779592037200928, -1.1864640712738037, -0.23892314732074738, -1.8307197093963623, -1.3894846439361572, 0.4308429956436157, 1.0449882745742798, -1.075324296951294, -0.40863773226737976, 0.4776780903339386, -0.5224950909614563, -0.5543418526649475, 0.05864739790558815, 0.34896034002304077, 2.1505818367004395, -1.6139938831329346, -1.4871371984481812, 0.3796347975730896, 0.8485768437385559, -0.2786281108856201, 0.5554987788200378, -0.34808987379074097, -0.2676796019077301, 0.8927844166755676, -0.8656686544418335, 0.14635755121707916, -0.1954147219657898, -0.548613429069519, -1.6764510869979858, 0.8730632066726685, -0.17632602155208588, 0.5410699844360352, -1.0810365676879883, -1.2360668182373047, 0.19198668003082275, 0.36524027585983276, 1.1918221712112427, -0.384050577878952, 1.541979432106018, 0.6410744190216064, -0.28539368510246277, 0.974113941192627, 1.3392531871795654, -0.2576662302017212, -0.6210886836051941, 1.1911981105804443, 1.105271339416504, -0.7235001921653748, 0.04504486918449402, 1.5026336908340454, 0.2911330759525299, -0.7354354858398438, 0.1746860146522522, -1.2181997299194336, -1.7702126502990723, 0.3423599302768707, -1.291926622390747, -0.06366560608148575, 0.09167362004518509, 0.618993878364563, 0.2551576495170593, 0.5756804943084717, -0.9231850504875183, 1.4022283554077148, 0.5458681583404541, 0.1645112782716751, 1.2961063385009766, 1.1290634870529175, -0.89105623960495, -0.2689329981803894, 0.4490346610546112, -0.09357805550098419, -0.7174736261367798, 0.5113034844398499, -0.555814266204834, 0.20458824932575226, 1.1208643913269043, 0.46335941553115845, 1.911125898361206, -0.8616700172424316, -0.6657007336616516, 0.4354112148284912, 0.25331488251686096, -0.36016273498535156, -0.058981288224458694, -1.3448458909988403, 2.5011942386627197, -0.145536407828331, -0.015280565246939659, 2.8004558086395264, -1.460383415222168, -0.9337475299835205, -1.3767285346984863, -0.19216495752334595, 0.33302173018455505, -0.3316226303577423, 0.35503119230270386, 0.4373715817928314, 0.10103202611207962, 1.0144661664962769, 1.2535032033920288, 0.8359367847442627, 0.24366013705730438, 0.5030327439308167, -0.9480170607566833, 0.008751663379371166, -0.9636178612709045, 0.0028497008606791496, 0.5203741788864136, -0.39850038290023804, -0.04782542958855629, 0.8000657558441162, -0.7260037064552307, -0.03589688614010811, 0.5375105142593384, -1.1796131134033203, 1.1433782577514648, 0.39131712913513184, 0.4407692849636078, -0.2607751786708832, 0.24347324669361115, 1.0807220935821533, -0.46341586112976074, -0.06887763738632202, -0.7498598098754883, 1.4042541980743408, -0.12201931327581406, 0.03510095179080963, -1.7743157148361206, -1.5508337020874023, -0.15716642141342163, 0.5543351769447327, -1.0740711688995361, -0.1810733526945114, -0.17910975217819214, 0.7455949783325195, -1.4941344261169434, -0.7100291848182678, -0.09315498918294907, 0.11703693866729736, -0.6636339426040649, -0.05016343668103218, -1.0201735496520996, 0.5566426515579224, 1.361998438835144, -1.2800296545028687, 0.38110116124153137, -0.48283323645591736, -0.3944609761238098, -0.7955494523048401, -0.7960796356201172, 0.5223686695098877, 1.6743923425674438, 0.015599052421748638, -0.7407523393630981, 0.026502113789319992, -0.1206890419125557, -0.27246880531311035, 1.210974931716919, 1.4835559129714966, 1.3542027473449707, 1.6493034362792969, 0.0052448236383497715, -0.4966241717338562, -0.9625416994094849, 0.27371132373809814, -1.0356017351150513, -0.9508849382400513, -0.3086128532886505, -0.2660521864891052, -0.1357884556055069, 1.9459203481674194, -0.667197585105896, -1.0883820056915283, -1.5683492422103882, -0.527295708656311, -0.47763702273368835, -1.7242580652236938, -0.05245066061615944, -0.9042267799377441, -0.5965892672538757, 0.12760254740715027, -0.6978892087936401, -0.9072138071060181, 0.6971469521522522, 1.5603926181793213, -0.44081223011016846, 0.8550897836685181, 0.31963226199150085, 0.19310708343982697, 0.8338058590888977, -1.3398712873458862, -0.4643859565258026, -1.8777962923049927, 0.13663022220134735, -2.1581873893737793, 0.7994956374168396, 0.6374335289001465, -0.033217184245586395, -2.5790226459503174, -0.9057678580284119, -0.11245976388454437, -0.8902718424797058, 0.863442063331604, 0.4229317903518677, 0.08062094449996948, -0.0024972036480903625, 0.023285960778594017, -1.0878865718841553, -0.34980419278144836, 0.8855447173118591, -1.13690984249115, 0.7386720776557922, 0.8444551229476929, 0.6334351897239685, 0.705084502696991, -0.1249721497297287, 0.22565537691116333, -1.526414394378662, -0.0076142470352351665, -0.7092499732971191, 1.3476179838180542, -0.6585628986358643, -0.1704644411802292, -1.3072106838226318, -0.7550265192985535, 1.1200246810913086, -0.950350284576416, 0.3050013482570648, 0.49230533838272095, -1.3081039190292358, -0.47081124782562256, 1.850818157196045, 1.0260844230651855, -0.003223283216357231, 1.3836597204208374, 0.9200862050056458, 0.24306048452854156, 0.7132663130760193, 0.060180600732564926, -0.06717538833618164, -0.6320303082466125, 0.30028262734413147, -1.0610597133636475, 0.2856205403804779, -0.37902170419692993, 0.9261139035224915, 0.22117620706558228, -0.9439218044281006, 0.48252421617507935, -0.7942065000534058, -3.0035901069641113, -0.73824542760849, 1.873260736465454, 0.07050760090351105, -1.6665141582489014, 0.6616430878639221, 0.9009604454040527, 0.5216504335403442, 0.33566415309906006, -0.35710379481315613, 0.6158243417739868, -0.11112959682941437, -0.4597443640232086, 0.2777147889137268, -1.1458085775375366, -0.46705976128578186, 0.3023826777935028, -0.7697145342826843, -0.06596959382295609, 0.7280362844467163, -1.7945266962051392, -0.9835051894187927, 0.4996449053287506, 0.20642872154712677, -0.6667524576187134, -0.8209594488143921, 0.31169283390045166, -0.2819526195526123, 0.9803400039672852, -0.5192808508872986, -0.5867493748664856, -2.0608716011047363, 0.4180453419685364, -1.4565616846084595, -1.2347224950790405, 1.019445538520813, -0.22191977500915527, 1.386361002922058, 2.0973548889160156, 0.06599627435207367, -0.37824153900146484, -1.1521122455596924, -0.7362709641456604, -0.8050792217254639, -1.6842560768127441, 0.6945599317550659, -0.08798901736736298, 0.6895768642425537, 0.43694648146629333, -0.6291801929473877, -1.7277071475982666, -0.7068082094192505, -0.04072674363851547, -0.8847606182098389, -0.28853514790534973, 1.129228949546814, -0.4083894193172455, 0.183019757270813, -0.37487146258354187, 0.1888365000486374, 0.5192781090736389, 0.37450456619262695, -0.8460756540298462, -0.08192505687475204, -0.024431569501757622, -0.6506687998771667, -1.3359955549240112, 0.5596664547920227, -0.14279165863990784, 0.2681099474430084, -0.9475221633911133, -0.9468995332717896, -0.8630818128585815, -0.36092409491539, 1.561028242111206, -2.528069257736206, 0.38833191990852356, 1.9026554822921753, -0.9591485261917114, -0.4441213607788086, 0.7197937369346619, -0.06793282181024551, -0.3998681306838989, 2.7608635425567627, 1.6036618947982788, -0.627919614315033, -0.6189988851547241, -1.501441240310669, 2.1782054901123047, 1.003589391708374, 0.2982137203216553, 0.6848016977310181, -0.8016815185546875, -1.3779253959655762, 1.8771507740020752, -0.20798921585083008, 2.3970870971679688, -1.216631531715393, -0.021933458745479584, -1.594665765762329, -0.3642275035381317, -1.5939399003982544, 1.4935959577560425, 1.065260410308838, 0.6395830512046814, -1.650749683380127, 0.16788391768932343, -0.21768158674240112, -0.06112686172127724, 1.3311196565628052, 0.9115157723426819, 1.5066553354263306, -1.5336663722991943, -0.26091721653938293, 1.4891148805618286, -0.08260966837406158, 0.55682373046875, -0.11768651753664017, 1.747728943824768, 0.9564008712768555, 0.3061639070510864, -1.9270683526992798, 0.4498840272426605, 0.8924022912979126, 0.2652739882469177, -0.7686991691589355, -1.1582841873168945, 0.7092544436454773, -0.36156243085861206, -0.05650641769170761, 0.8074440360069275, -0.3011893630027771, 1.3221126794815063, -0.33442243933677673, 0.10131001472473145, -0.9894979000091553, 0.2207462638616562, 1.140039324760437, -0.13893499970436096, -1.0049827098846436, -0.7141567468643188, -0.24245017766952515, -2.1341612339019775, 0.6699923276901245, 0.15162478387355804, -1.545658826828003, 1.4849989414215088, -0.4625494182109833, 1.4339747428894043, 0.827483594417572, -1.8947340250015259, 1.5565989017486572, -1.3451913595199585, 0.48356637358665466, 0.11863773316144943, -0.45940497517585754, 1.7554268836975098, 1.0003819465637207, 0.554490327835083, 0.14868775010108948, -1.0191736221313477, -0.22050565481185913, 0.5858874320983887, -0.44073405861854553, 0.14385710656642914, -0.32320311665534973, -0.5195935368537903, 1.2382901906967163, 0.17705640196800232, -1.7811754941940308, 0.5694164633750916, -0.5175754427909851, 0.27963340282440186, 0.14458328485488892, -0.1718892902135849, 0.35023269057273865, 1.5263237953186035, -1.2117244005203247, -2.8158810138702393, -1.9835014343261719, -0.06221886724233627, 1.7119840383529663, -0.65800940990448, -0.4784321188926697, -2.4961066246032715, 0.5304191708564758, 0.9810379147529602, 0.9983857870101929, -2.476679801940918, -0.255559504032135, -0.14158077538013458, -0.9088705778121948, 0.05158183351159096, -0.4116058945655823, -0.13930392265319824, -0.9976105690002441, -1.6112653017044067, -1.3336626291275024, 0.2733394503593445, -1.415228009223938, -1.779677391052246, 0.7371198534965515, 0.1463243067264557, 0.6159840226173401, -0.5166544318199158, -0.9904898405075073, -0.6749465465545654, -2.7157397270202637, -0.6876828074455261, -1.8696402311325073, -0.8689895868301392, 0.1554798036813736, 0.14734762907028198, 0.16347160935401917, -1.076587200164795, 1.355875015258789, 0.022782107815146446, -0.7449932098388672, -0.9083563685417175, 0.7344509363174438, -0.8190122246742249, -1.1286993026733398, 0.005376604851335287, -0.7687287926673889, 1.5685436725616455, 0.5723563432693481, -1.6616700887680054, 0.6124668717384338, -2.7834975719451904, -0.6587724685668945, 0.9883019924163818, -2.2151527404785156, 1.2246382236480713, 0.5527962446212769, 0.07002678513526917, -0.9978464841842651, 0.722683310508728, -1.2281644344329834, 0.0034736287780106068, -2.0729258060455322, 0.007022043690085411, 1.576528787612915, 0.15715272724628448, 0.612230658531189, -1.3650418519973755, 0.12309318780899048, -0.9904654026031494, -0.40624353289604187, 0.6886159181594849, -0.7853609919548035, 1.1088769435882568, 0.914139986038208, 1.9073072671890259, -0.0792628824710846, 1.1829034090042114, -2.535902738571167, 0.39231905341148376, -1.9706604480743408, -1.0102006196975708, -1.1275359392166138, 0.5250394940376282, 0.4383881390094757, -0.48431333899497986, -0.06191825121641159, -0.7921218276023865, -1.31728196144104, 0.5552007555961609, -0.08057548105716705, 0.9214001893997192, 0.41499146819114685, -1.5371092557907104, 1.6107556819915771, 0.737657904624939, -0.5596214532852173, -0.33014658093452454, -0.6034364104270935, 0.2792339324951172, 0.15205004811286926, -0.2541770040988922, 0.7272992134094238, -0.30803635716438293, -0.4665572941303253, 1.8128353357315063, -0.9356997013092041, 0.7425975203514099, 0.37604835629463196, 0.9412291646003723, -1.0821418762207031, 0.2860892713069916, 2.6906328201293945, -0.2812032699584961, -0.6347672343254089, 1.1394050121307373, -0.39303886890411377, -1.201699137687683, -0.5304664969444275, -0.8471624851226807, -0.1034758985042572, -1.070113182067871, -0.2857564389705658, 1.0421509742736816, 0.2616196274757385, -0.7047603130340576, -0.35276955366134644, -1.2469621896743774, -0.6801612973213196, -0.2851223349571228, -0.3352433443069458, 1.0824846029281616, 0.33774715662002563, -0.13824446499347687, -2.5203094482421875, 0.5473991632461548, 0.771403968334198, -0.6005917191505432, 0.17165090143680573, -0.45842263102531433, 0.42671656608581543, 1.092839002609253, -0.5371608734130859, -1.0697481632232666, 0.4106603264808655, 0.5155162215232849, -1.6725431680679321, -0.06127281114459038, 0.6792059540748596, 0.9050077795982361, 1.2667717933654785, -1.088559865951538, 0.23084241151809692, -0.5121811032295227, -0.018166434019804, -0.49568817019462585, 0.8335148096084595, 0.333359956741333, -0.07152935117483139, -0.7912254333496094, -0.6673294901847839, -1.0379539728164673, 0.4936542212963104, 0.26100966334342957, 0.03483979403972626, -2.610578775405884, -0.9541787505149841, 0.7321429252624512, 1.3017363548278809, -1.3385109901428223, 0.07703178375959396, -0.08437180519104004, -0.3848218619823456, 1.1924093961715698, 0.9888678789138794, 0.5845202803611755, 1.3443951606750488, 0.3953307271003723, 1.4768366813659668, -0.01549439039081335, -0.7310823798179626, 0.20943567156791687, 1.9354588985443115, 1.3498998880386353, -0.9468262791633606, 1.05191969871521, -0.47282496094703674, -1.5781904458999634, 0.34841206669807434, 0.8035861253738403, -1.1791229248046875, 0.28345611691474915, -0.4883222281932831, -2.3105618953704834, 0.11159218847751617, -0.9287095069885254, 0.020275546237826347, -0.5628222227096558, -0.3983486592769623, 0.1031683012843132, 0.6103786826133728, 0.17945680022239685, -1.2893255949020386, -1.7013914585113525, 0.4410054087638855, -0.7455326318740845, 0.29801782965660095, 0.3259178698062897, -0.9327795505523682, -0.033263251185417175, 0.8994454145431519, 1.394060730934143, 0.817497730255127, 0.2689022123813629, 1.55963933467865, -1.8234905004501343, 0.053958021104335785, 0.4732506573200226, 1.7003358602523804, -1.0529731512069702, -0.5973424911499023, -1.4384328126907349, -1.2884161472320557, -0.01329986285418272, 0.15552091598510742, -0.3289167284965515, 0.06511389464139938, 0.6059143543243408, 0.8327681422233582, 0.4340405762195587, 1.4830758571624756, -0.5474783182144165, 0.39186036586761475, -0.35658156871795654, 1.2944693565368652, -0.6116084456443787, -0.6702373623847961, 0.4882955551147461, 0.3478530943393707, -1.2158013582229614, 0.21615926921367645, 0.7972524762153625, -0.9631754159927368, -2.2030773162841797, -0.9453762173652649, -0.6568838953971863, -0.9540506601333618, -0.10047993808984756, 1.5624310970306396, 0.794293224811554, 0.32586804032325745, 0.5724502801895142, -0.39739474654197693, 1.4564231634140015, 0.0320449098944664, -2.0770833492279053, 0.3278075158596039, -0.45899391174316406, -0.3889346420764923, 1.8663064241409302, -0.8815078139305115, 0.1193419098854065, -1.8423185348510742, -0.4022551476955414, 0.41036805510520935, -0.4794183671474457, -0.22863686084747314, -0.7531827092170715, -1.8746134042739868, -0.3257874846458435, -0.018939929082989693, -0.8806490898132324, -1.8867920637130737, -0.33887091279029846, 0.025376036763191223, -0.06198359653353691, 0.05622730404138565, 0.06473363190889359, 1.7397725582122803, -2.0308077335357666, 0.013281266205012798, -0.6259697079658508, 1.3309193849563599, -1.3848302364349365, 0.2564579248428345, 0.968440592288971, 0.2830292880535126, 0.49041473865509033, -1.4014520645141602, 0.14218343794345856, 0.22174172103405, -0.9754757285118103, 0.3197857141494751, -1.1688737869262695, 0.5311002135276794, -0.161349818110466, -0.9128945469856262, -0.7529653310775757, 1.0722057819366455, 0.4997849762439728, 1.7319543361663818, 0.8983579277992249, 1.2158347368240356, -1.1506178379058838, -0.25715047121047974, 0.00951217021793127, 0.9936071038246155, -2.9088432788848877, 0.5261889100074768, -0.2514069974422455, -0.9131280779838562, -0.46506237983703613, -1.9898713827133179, -1.9359406232833862, -0.7403922080993652, -0.7683884501457214, -0.1400260180234909, -0.30951201915740967, 0.378937691450119, 1.730522632598877, 0.05814095214009285, -1.5003094673156738, -0.7940307259559631, -1.7796392440795898, 0.6663932800292969, -1.1161963939666748, -2.1675329208374023, 0.9531634449958801, -0.4073755443096161, 0.5332685112953186, -1.3045451641082764, 1.368928074836731, -0.8903627991676331, -2.1037757396698, -0.7766767144203186, 0.08932038396596909, 0.23128753900527954, 1.4154661893844604, -0.6862762570381165, -0.8088138699531555, -0.3516280949115753, 1.9935200214385986, 0.15075840055942535, 0.6475164890289307, -0.3961191475391388, 0.38590100407600403, 1.3431954383850098, -1.1718451976776123, -1.5895870923995972, -1.1309016942977905, -1.3198028802871704, -1.469794511795044, -0.17326350510120392, 0.9336401224136353, 0.40428781509399414, -0.5245762467384338, 0.7341498136520386, -0.9402042627334595, -2.5111894607543945, -1.4357411861419678, 0.9686512351036072, -1.886920690536499, 0.9910857677459717, -2.310821771621704, 0.7171205282211304, 0.545021116733551, -0.1022520437836647, -0.2320508062839508, 0.25763174891471863, -0.3003499507904053, -1.3961330652236938, 1.3320552110671997, -0.9723324179649353, 1.2626687288284302, -1.6338557004928589, 0.05918693169951439, 0.7032530903816223, -0.5530577301979065, -1.4126684665679932, 0.12153078615665436, -1.5493603944778442, 0.2080049365758896, -1.3324187994003296, -0.3491976261138916, -0.4283911883831024, 0.3324763774871826, 0.2654554843902588, 0.13654430210590363, -0.694442868232727, 0.22771242260932922, -0.41234615445137024, -0.7733526825904846, -0.015062312595546246, 0.9743863344192505, -1.9639111757278442, -0.7175238132476807, 2.0694754123687744, -0.1958964765071869, -1.8179036378860474, -0.4705010950565338, 0.4031822383403778, 1.151245355606079, -0.13116832077503204, 0.5849018692970276, -1.6000628471374512, 1.1416453123092651, -1.421384572982788, -0.24653899669647217, -1.8749933242797852, -0.20759309828281403, -0.384573757648468, -1.5163241624832153, -1.6476119756698608, 0.9321223497390747, 0.08803775906562805, -2.0260350704193115, -1.5490397214889526, 0.03671093285083771, 0.15389518439769745, -0.08456077426671982, -0.14050599932670593, 0.9875814318656921, -0.001732606440782547, 0.6977920532226562, -1.01316237449646, -0.33074241876602173, -0.4903481900691986, -0.1800697147846222, -0.38029369711875916, 0.716882050037384, -0.05573492497205734, -1.2396289110183716, 0.5146986246109009, 1.2923072576522827, -0.8224195837974548, 2.043086051940918, 0.40365472435951233, 0.25656551122665405, -0.5839598178863525, 1.1603904962539673, 0.13931205868721008, -1.011565089225769, 1.5383334159851074, -0.8019360899925232, 1.0090594291687012, 0.7742464542388916, 0.2061159312725067, -1.142022728919983, 1.0119363069534302, -0.4416353702545166, 0.024147113785147667, 0.14294511079788208, 1.8895999193191528, 2.0684397220611572, 0.5545940399169922, 0.8014909029006958, 0.2714902460575104, 1.8024139404296875, 0.6655887961387634, 1.2009530067443848, 2.8925957679748535, 1.0662871599197388, 0.38813450932502747, -0.6215149164199829, -1.6285531520843506, 1.4850139617919922, -0.8887937664985657, 0.083026222884655, -1.1285024881362915, -1.215946078300476, -0.030017441138625145, 1.469240427017212, 0.2703068256378174, -0.574852705001831, -1.8876301050186157, -2.603482723236084, -0.7887791991233826, -1.947876214981079, 1.9041179418563843, -0.26907452940940857, -0.42228779196739197, -0.09408438950777054, 1.3197318315505981, 0.3946191668510437, -1.9256882667541504, 1.758396029472351, -2.500776529312134, -0.6869086623191833, 1.735129714012146, 1.7306718826293945, 0.8226414918899536, -0.2611607313156128, -0.4399394989013672, -0.3817654252052307, -0.9025191068649292, -0.20236368477344513, 0.007134349551051855, -1.7932878732681274, -0.23395437002182007, -0.1783570498228073, 0.025722689926624298, -0.10288234055042267, 0.8303554058074951, 1.6837995052337646, 0.9590053558349609, 1.1164592504501343, -0.36191341280937195, -1.5916812419891357, 1.519607424736023, -0.5986608862876892, -0.13313154876232147, -0.3730437457561493, -0.0372428297996521, 0.4362023174762726, 1.3884855508804321, 0.8557490706443787, 0.4939228892326355, -0.32891151309013367, -0.6371113657951355, 0.317082941532135, -0.9605541229248047, -1.2338802814483643, -0.036904338747262955, 0.16148455440998077, -0.4203106462955475, -0.05084043741226196, -0.9739345908164978, -0.1302662044763565, -1.3745089769363403, -0.5196132063865662, -1.5444012880325317, -1.2655977010726929, -0.7182182669639587, 0.9172580242156982, -1.0597690343856812, -0.23744870722293854, -0.4216385781764984, -0.565187931060791, -1.2221895456314087, 0.2371559888124466, -0.4716435372829437, 0.12491068989038467, -0.8978980183601379, 0.3655901551246643, -0.5074553489685059, 0.4110087454319, 0.498660147190094, -0.9018749594688416, 0.33055630326271057, -1.099219799041748, 0.9311176538467407, -1.293312907218933, -1.101181983947754, -1.666981816291809, 1.3305588960647583, 1.1518620252609253, 1.623901605606079, 0.2532198429107666, 0.3610747158527374, -1.0263036489486694, 1.3080825805664062, 1.2703312635421753, -0.5515490770339966, -1.1207219362258911, -0.31249070167541504, -1.4917856454849243, 0.37131381034851074, -2.2114551067352295, 1.0018861293792725, -0.3335430324077606, 0.34792590141296387, 0.12656448781490326, 0.0939464196562767, 1.1348261833190918, -1.3951339721679688, 1.0893902778625488, -0.7583175301551819, -0.5933839082717896, 1.414480209350586, -0.46033626794815063, -0.04849648475646973, -1.892930507659912, -1.334256887435913, -0.37756484746932983, -0.6605538129806519, 0.4116387367248535, 1.4113465547561646, 0.16953270137310028, -0.45782795548439026, 0.0992632508277893, 1.588722586631775, -2.089128255844116, -0.19800381362438202, 0.2858445942401886, -0.1342024803161621, 1.3660370111465454, 0.803634762763977, 0.069198839366436, 0.5507972836494446, 0.5940165519714355, -1.1600667238235474, -1.2981536388397217, -0.5930973887443542, 0.5075693130493164, 0.46418502926826477, -0.6280273795127869, -0.3252411484718323, 0.6912265419960022, -0.2870030105113983, -0.5625783801078796, -0.6150566339492798, -0.48493990302085876, 0.8896198868751526, 1.1735676527023315, 0.4691765010356903, -0.19857487082481384, -1.4743965864181519, -0.22406697273254395, -0.9891878366470337, 0.02638867497444153, 0.3944041132926941, -0.8601744771003723, -0.5225238800048828, -1.6015056371688843, 2.7194037437438965, 1.1656068563461304, -1.954224705696106, 2.6903507709503174, -0.4643622636795044, -1.0295096635818481, 0.6464858651161194, 1.1018867492675781, -1.2280654907226562, 1.5184537172317505, -1.0294055938720703, -0.439035564661026, -0.9278690218925476, 0.6325203776359558, -0.15285973250865936, -0.7065169811248779, 0.40911442041397095, -0.6578607559204102, 0.37435033917427063, 1.2685532569885254, 0.637272298336029, 0.8150296211242676, 0.6762864589691162, 0.18563206493854523, -0.4202645421028137, 2.122525453567505, 0.6854562759399414, -0.7927344441413879, -2.719878911972046, -2.2225406169891357, -0.11455778777599335, -1.0884257555007935, 0.277323454618454, -0.9308056235313416, -1.1511412858963013, 0.39180293679237366, -0.48821884393692017, 0.675057053565979, 0.01956084743142128, -0.8213362693786621, 1.0426594018936157, 1.255354881286621, -0.09511921554803848, 1.4049311876296997, -0.18820136785507202, -0.2787224054336548, -0.30141234397888184, 0.5375804901123047, -0.23185725510120392, -2.92667555809021, -0.352046936750412, -0.20886439085006714, 0.5869615077972412, -2.02630615234375, -0.023074980825185776, 1.5328450202941895, 0.020454538986086845, -0.9084293246269226, 0.2302693873643875, 0.3004365861415863, -1.050002098083496, -0.10144461691379547, -0.12290695309638977, -0.33245256543159485, 0.3028707206249237, 0.3699995279312134, -1.9635541439056396, 0.9647099375724792, 0.18873514235019684, 0.5347286462783813, 0.7673609256744385, -1.4558906555175781, -0.9405739903450012, -1.0576528310775757, 0.043381467461586, 0.9606733322143555, 0.9247934818267822, -0.13824462890625, 0.37066566944122314, -0.26054731011390686, -0.5279484987258911, 0.9640917778015137, 0.18842755258083344, 1.1517256498336792, -0.8838152289390564, 1.161420464515686, 0.48701193928718567, -1.0035609006881714, -1.7026467323303223, 1.127829909324646, -1.447892427444458, 0.4199126958847046, -1.9604166746139526, 0.25175416469573975, 0.41572126746177673, -0.8871261477470398, -0.5717029571533203, -0.91194087266922, 0.2814849019050598, -0.28246456384658813, 0.5359000563621521, -0.2692476511001587, 1.090950608253479, 0.9723126292228699, -0.8329014182090759, -0.6623519062995911, -0.3075716197490692, -0.4211556315422058, 0.7383592128753662, -0.5669972896575928, 1.6189442873001099, 0.5407124757766724, 0.17731419205665588, -0.4435095191001892, 0.9430931210517883, 1.4075419902801514, 0.4110024869441986, 0.5510386228561401, -0.604790985584259, -0.6305540204048157, 0.8478214144706726, 0.09879406541585922, -1.107073187828064, 0.37550947070121765, -0.024052871391177177, -1.9445124864578247, 1.0647342205047607, 0.8823406100273132, 0.30946630239486694, 0.5432239770889282, -0.09666641056537628, 0.9271582961082458, 0.1721879094839096, -0.8587985634803772, -1.0410995483398438, 1.3018730878829956, -1.7482720613479614, -0.7448973655700684, -0.4177663028240204, -1.7694445848464966, 1.7774014472961426, -0.8994919657707214, -0.05640313774347305, 1.124954342842102, -0.93455970287323, -1.8218259811401367, 0.2304535210132599, -0.37988507747650146, -0.3745552897453308, 0.4639861285686493, -0.02175208553671837, -0.7300275564193726, -0.4650552272796631, 0.8287339210510254, 0.20809979736804962, 0.7050275802612305, -0.629667341709137, 0.5789791345596313, 0.014217608608305454, 0.048475541174411774, -0.9720389246940613, -2.193341016769409, 2.5258114337921143, -0.815386176109314, -0.17854757606983185, 0.41120901703834534, 0.707099974155426, -0.4033251106739044, 0.43299490213394165, -1.3129503726959229, -0.9703352451324463, 0.9349057674407959, 0.9862631559371948, -0.405940979719162, -0.24161939322948456, -0.3059265911579132, 0.0008893460617400706, -0.06265193969011307, -0.23250775039196014, 0.6062226891517639, 2.2152247428894043, 1.7630640268325806, -0.5401616096496582, 1.3587003946304321, 1.3014488220214844, 0.5843588709831238, -1.0862703323364258, -0.2740239202976227, -1.5846270322799683, -0.7685571908950806, -0.20392337441444397, -0.37163856625556946, -0.1805305927991867, -0.7273135781288147, 1.4900224208831787, -0.14624451100826263, 1.3637412786483765, 0.897092342376709, -1.0414338111877441, -0.4506518840789795, -0.054435957223176956, -0.9501073956489563, 0.6196670532226562, -2.1528215408325195, -1.0292658805847168, -0.2781311869621277, -1.159206748008728, -1.5466787815093994, 0.6570140719413757, -0.3402687609195709, -0.0950012281537056, 0.2442711591720581, 0.5929082632064819, 1.375432014465332, -0.08649114519357681, 0.573701024055481, 0.18862660229206085, -1.3100860118865967, -1.5270103216171265, -0.06634503602981567, 1.5280377864837646, -0.6801896691322327, -1.3121610879898071, 0.4762842357158661, -0.14778929948806763, -0.6170552968978882, -0.2604726254940033, 1.1043901443481445, -0.07717610895633698, -1.7154462337493896, -0.42607781291007996, -0.6985852122306824, -0.22959640622138977, 0.6340469717979431, 1.1563401222229004, 0.8603131771087646, -1.2059754133224487, 0.3252760171890259, 1.6832488775253296, -0.5858300924301147, -0.33543455600738525, -1.0197749137878418, -1.1682685613632202, -0.4316370189189911, -2.3003578186035156, -0.5797696709632874, -0.19239607453346252, -0.04642689600586891, -0.7964257597923279, 0.5823060870170593, 0.13237830996513367, 0.9593547582626343, 1.099109172821045, -1.3031827211380005, 0.5551227331161499, -0.28035515546798706, -1.4911463260650635, 0.08815654367208481, 1.1388717889785767, 0.38607507944107056, 0.7085149884223938, -0.6833134293556213, -0.2606544494628906, -0.8410813212394714, -0.4258284568786621, -0.5965058207511902, -1.2496408224105835, 0.11798935383558273, -0.811683177947998, -1.237252950668335, 1.7757337093353271, 1.5980572700500488, 0.035572946071624756, 1.95863938331604, 0.6392178535461426, -0.7006980776786804, -1.583266258239746, 1.12492835521698, 0.5336670875549316, 0.43097615242004395, -1.615917682647705, 0.6365473866462708, -1.8074686527252197, -1.1076010465621948, 0.6225149631500244, 0.8045837879180908, -0.31555718183517456, 0.8818077445030212, -2.768071413040161, 0.7568577527999878, 0.46468624472618103, 1.6497504711151123, 1.1752288341522217, 0.3650394380092621, -0.5482591390609741, -2.569061040878296, 1.227080225944519, -1.223763346672058, 1.9938219785690308, 1.68217933177948, -0.8467908501625061, 0.8209500312805176, 0.32305026054382324, -0.023057589307427406, -1.4298392534255981, 1.461267352104187, -1.1481740474700928, 0.8752509355545044, -0.4721490740776062, -0.32673755288124084, 2.376695394515991, 0.07276005297899246, -0.26647424697875977, -0.6534379124641418, -0.34824785590171814, 2.1774189472198486, -2.2881875038146973, 1.602724552154541, 0.3010426163673401, 0.5377821922302246, -0.10760580003261566, -1.0730394124984741, 1.1158485412597656, -1.13117516040802, -0.40068405866622925, 0.49734458327293396, -0.1763773262500763, -0.19283805787563324, 0.1662374883890152, 1.7043936252593994, 0.5559305548667908, 1.589457392692566, -2.0291011333465576, 0.7533127069473267, -1.5033962726593018, 0.44704383611679077, 1.0064697265625, 0.08976873010396957, 1.9231417179107666, -0.9984071850776672, 0.21837209165096283, 0.44380679726600647, -0.4463900625705719, 0.9816523194313049, 0.15463002026081085, 0.8723017573356628, 0.1289897859096527, -1.7208853960037231, 2.23844575881958, -0.26658135652542114, 0.4429182708263397, 0.996910810470581, -1.0990910530090332, 0.01020316407084465, 0.2754749059677124, 0.5462556481361389, -0.07719913125038147, 0.6016778945922852, -0.630137026309967, -1.4376243352890015, -0.44259366393089294, 0.2204650640487671, 0.6524670720100403, -0.20243658125400543, -0.40496426820755005, -1.6631516218185425, 0.24475212395191193, -0.9294300079345703, -0.0858507975935936, 1.7698664665222168, 0.3399586081504822, -0.88736492395401, 1.3640321493148804, -1.1572726964950562, -0.1408964842557907, 1.152263879776001, 2.136859178543091, 0.71865314245224, 0.9619132876396179, -0.8676418662071228, 0.497226744890213, 0.048846837133169174, 0.7398731708526611, -0.20743803679943085, -1.1943787336349487, -1.4051119089126587, 1.2933052778244019, 0.3801751136779785, -0.44843581318855286, 0.255587637424469, -1.1628412008285522, 0.0866350382566452, 0.06901666522026062, 0.07042383402585983, 2.881877899169922, -0.6920127272605896, -0.6363140940666199, -0.0011153476079925895, -0.26562294363975525, -0.7222275137901306, -2.4824862480163574, -0.6688608527183533, -2.0932023525238037, 1.2532806396484375, -0.5332469940185547, 1.494079828262329, 2.7377588748931885, -0.4378657639026642, 0.9680408835411072, -1.2513985633850098, -0.33383744955062866, -0.5005141496658325, 1.0622901916503906, -0.33437487483024597, -1.3348755836486816, 0.5395192503929138, -0.6740989089012146, 0.3469432294368744, 0.49124205112457275, -0.7011895775794983, 0.6877465844154358, -1.0687671899795532, -2.5194344520568848, -0.2949010729789734, 1.971794843673706, 0.2743450403213501, 0.2628147304058075, 0.9683594703674316, -1.0437198877334595, -1.088695764541626, 0.4554319381713867, 1.008865237236023, -1.0244766473770142, 0.6480019688606262, -0.09446722269058228, 0.6233454346656799, 0.7282776832580566, -0.09751105308532715, -1.0655113458633423, 0.17119775712490082, 0.6368088126182556, 0.07243221253156662, -2.778489828109741, -0.16251610219478607, -1.357467532157898, -0.6571880578994751, 0.3410854935646057, 1.1859357357025146, -0.6247391104698181, -0.9090546369552612, 0.4222264885902405, 0.49238017201423645, -0.2049722671508789, -0.4092814326286316, -0.09746529906988144, -1.257561445236206, 0.8249294757843018, 0.2572392225265503, -0.2478748857975006, 2.6868064403533936, 0.29797378182411194, -0.2447078675031662, 0.952061116695404, 1.651928424835205, 0.307808518409729, -0.2736501693725586, -0.7333569526672363, -0.2882518172264099, 0.006440720986574888, 1.8384125232696533, -0.59774249792099, 0.3756878674030304, 0.39557260274887085, 1.0765275955200195, 0.645679771900177, -0.3659900426864624, 0.5610449314117432, -0.30359750986099243, 0.818270742893219, -0.8951530456542969, -0.924808919429779, -0.08319146931171417, 0.6063848733901978, -0.5832154154777527, 1.0826665163040161, -0.3579840064048767, -0.4539273977279663, 1.4539899826049805, 1.404928207397461, -0.5108765959739685, -1.1136029958724976, -0.9366589188575745, 0.029970744624733925, 0.45262590050697327, 0.8925375938415527, 0.44132471084594727, -0.4512478709220886, -0.253182053565979, 0.40623897314071655, -1.1981459856033325, 0.6388282775878906, -2.334568977355957, -1.3464088439941406, 0.17555178701877594, -1.2477153539657593, -2.3287699222564697, -0.6696677803993225, 0.9959072470664978, -0.5795022249221802, 0.1967138648033142, -0.18848921358585358, 0.4215489625930786, 1.718259334564209, -0.17595085501670837, -1.485244631767273, -0.8171780109405518, 0.6229219436645508, 0.10825270414352417, 0.6638902425765991, 2.5101757049560547, -0.9256070256233215, 0.9783512949943542, -1.8718972206115723, 0.811535656452179, -0.663216769695282, 0.08248583972454071, -0.20543266832828522, 1.1345570087432861, 0.4582972228527069, -1.2286170721054077, 0.3607439398765564, -0.6785953044891357, 1.4687199592590332, -0.6188186407089233, -1.1424397230148315, 1.7553502321243286, -0.9499560594558716, -0.7011619210243225, 0.5712668895721436, -0.6357913613319397, -0.6085753440856934, -0.6235239505767822, 0.32951316237449646, -2.9338841438293457, 1.3034838438034058, -0.849185585975647, -0.877386748790741, -0.7889834046363831, -0.8210574984550476, -0.33613231778144836, -0.34873104095458984, -0.7861424088478088, -1.648358941078186, -0.0041479384526610374, -1.0598653554916382, -2.5599703788757324, 0.8731865286827087, 0.6419434547424316, -0.3413655161857605, -0.9874356389045715, -1.1592499017715454, -0.2615601420402527, -0.5705143213272095, -1.573347806930542, -0.09857496619224548, -0.8162881731987, 0.8402760624885559, 0.5243321657180786, 0.11639083921909332, -0.9627518057823181, -0.4099498391151428, 1.0822083950042725, -0.9126632213592529, -1.4614582061767578, 1.6813786029815674, 0.1271907538175583, 1.2866742610931396, -1.126354455947876, 0.31671133637428284, 1.8005013465881348, -1.596571922302246, -0.2693096697330475, -0.37310972809791565, -0.66087806224823, 0.6701927781105042, -0.46134379506111145, -0.49322474002838135, -0.015693217515945435, 0.6823620200157166, 2.2358899116516113, 0.7055665254592896, -0.23881462216377258, -1.4103437662124634, -0.8228821754455566, -0.04505420848727226, -0.8694155216217041, 0.6898375153541565, -0.4935418963432312, 0.7343339323997498, -0.32153868675231934, 0.7260039448738098, 0.16990543901920319, 0.479640930891037, -0.25565972924232483, 0.976631760597229, 0.8145696520805359, 0.7501225471496582, 0.9915466904640198, 0.7912600040435791, -0.46602028608322144, 1.3954869508743286, -0.3419797122478485, -0.8579617142677307, 1.8464826345443726, 0.970963716506958, 0.5288676619529724, 0.8315114974975586, -0.8055745363235474, 0.5032346248626709, -1.7407891750335693, 0.4059414565563202, 0.8480053544044495, -0.6601758599281311, 0.25876274704933167, -0.4660545289516449, -0.7360742092132568, -1.0886503458023071, 0.06927729398012161, -0.5623613595962524, -1.5660698413848877, -0.5767837166786194, 0.7967121601104736, 0.34261590242385864, -0.785620927810669, 0.5401728749275208, -1.1819279193878174, 1.0872242450714111, 1.119879961013794, 0.01724165864288807, 0.3864959478378296, 0.221351757645607, -0.2519192099571228, 1.4264658689498901, -0.9192457795143127, -0.6529731154441833, -0.3153093457221985, -0.03206745535135269, -0.4455934762954712, -0.6028285622596741, 0.16206307709217072, 1.19239342212677, -2.5753962993621826, 0.8721182942390442, 0.833182156085968, 1.3255220651626587, -0.7822009921073914, -0.62502521276474, 0.28990790247917175, -0.8048353791236877, -1.44011652469635, -0.015462060458958149, -0.3613283932209015, 0.20226149260997772, 0.5685592293739319, -1.5062735080718994, 0.724583625793457, -0.3254273533821106, -0.32866185903549194, 1.4634394645690918, 0.14998474717140198, 1.2189295291900635, 1.080520510673523, 0.2450578808784485, 0.8072514533996582, 0.9132686257362366, 0.9937981963157654, -1.240954041481018, -0.8081770539283752, -0.3917681872844696, -0.9408358335494995, 0.9744572639465332, 0.3920499086380005, 0.6381007432937622, -0.5547946691513062, -2.442610025405884, 1.804777979850769, 2.9146039485931396, -0.8850926756858826, -1.0832107067108154, -0.8212611079216003, 0.04389762505888939, 0.6394962072372437, 0.19935080409049988, -1.55777108669281, -0.16953305900096893, -0.8317974209785461, 0.31927424669265747, 1.0764786005020142, -0.788536548614502, 0.4263734221458435, 0.6024455428123474, -0.24619878828525543, -1.0133014917373657, -2.0628437995910645, 1.2629960775375366, -0.3299042880535126, 0.6944770216941833, 0.9668103456497192, -0.731740415096283, -0.4032280445098877, 0.5197668671607971, 0.47770169377326965, -1.5912572145462036, 1.3492865562438965, -1.0352035760879517, 0.1243835911154747, -0.48453202843666077, -2.029384136199951, -0.5348795652389526, 0.11232997477054596, 1.0231847763061523, -1.1533399820327759, -2.0593976974487305, -0.39073646068573, 2.221407890319824, -0.6605632305145264, -0.2833554744720459, 0.4640938341617584, -0.7396852970123291, -0.10390156507492065, 0.024366816505789757, -0.007305356208235025, 0.6755040884017944, -0.33869585394859314, 0.4549708962440491, 0.5761181712150574, -0.8474093675613403, 1.3812237977981567, 0.5063451528549194, -0.4985331892967224, 1.1692512035369873, -0.9575952887535095, -1.7483052015304565, 0.10848955810070038, -1.6319702863693237, 0.06516602635383606, -0.9944213032722473, 0.23441527783870697, -0.49982336163520813, -0.642246425151825, 0.18615640699863434, -0.511003315448761, 0.1530994474887848, -1.4429579973220825, 0.3367353081703186, 0.6106969118118286, 1.1089125871658325, 0.6748952865600586, 0.7577295899391174, -0.12660884857177734, 0.009300070814788342, -0.06420882791280746, -0.4680008292198181, -1.9876364469528198, -0.016052164137363434, -2.440840244293213, 0.35912638902664185, -2.665778398513794, 2.4187586307525635, 0.31478846073150635, -0.7293755412101746, -0.5984535217285156, 0.47890686988830566, 0.9208233952522278, -0.743460476398468, -0.4291094243526459, 0.7716185450553894, -1.4878817796707153, -1.2160825729370117, -0.17593543231487274, 0.6913554668426514, 0.0011333636939525604, 0.9107790589332581, 1.3886759281158447, 0.8939067125320435, 2.2903025150299072, 1.342368483543396, 0.5062525272369385, 2.9731786251068115, 1.0852246284484863, 0.5808337926864624, -1.9803818464279175, -1.8920395374298096, -0.00587483448907733, -1.92621910572052, 0.3845987617969513, 0.12991422414779663, -1.5606883764266968, 0.08904042094945908, -1.1576426029205322, -0.08033975958824158, -0.004555453546345234, -0.21913890540599823, 0.8935297131538391, -1.1115280389785767, 0.0409667044878006, -0.7403644919395447, -0.9860279560089111, 0.6337296366691589, -1.1255244016647339, 0.004494714085012674, -0.21837492287158966, -1.294149398803711, -1.2267369031906128, 0.02269442193210125, -0.12061648815870285, -1.544700026512146, -0.8309610486030579, -1.5269397497177124, 0.6918451189994812, -0.6278622150421143, 0.19959145784378052, -1.2887086868286133, 1.215423345565796, -1.0679768323898315, -1.4276243448257446, 1.1260671615600586, 0.3902570605278015, 0.09799037873744965, 0.6260900497436523, 1.2383146286010742, 0.6089515089988708, -0.12406621873378754, 0.7693847417831421, 0.33996763825416565, 0.4939653277397156, 1.246683955192566, 0.33576127886772156, 0.17887116968631744, -0.1599574089050293, 1.2422116994857788, 0.22047393023967743, -0.3752194046974182, -0.3352595865726471, 1.2356878519058228, -0.5116279721260071, 0.5881426334381104, -1.3094760179519653, 0.8109911680221558, 1.972590684890747, 1.1272081136703491, 1.432037353515625, -0.16637037694454193, -0.10583455860614777, 0.9238857626914978, 0.010746447369456291, 0.8355887532234192, 0.19524401426315308, 0.4469431936740875, -0.7238180041313171, 0.630351185798645, 0.34790265560150146, -1.3474531173706055, 1.2742078304290771, 1.5951176881790161, 0.323055624961853, 1.3466272354125977, -0.49619364738464355, -0.07453517615795135, -1.1756149530410767, 1.2997151613235474, 0.27059757709503174, -0.29599621891975403, 1.3300822973251343, 2.9785287380218506, 0.30611079931259155, 0.08493832498788834, 0.7543480396270752, 0.4219423234462738, 0.5961307287216187, 0.8863844275474548, -1.1622847318649292, -0.016790760681033134, -0.3788232207298279, 1.3922832012176514, 1.3234875202178955, 0.07856162637472153, -1.3879914283752441, -0.2555355727672577, -1.1248903274536133, -0.19314348697662354, 0.4053223431110382, -0.6742268800735474, -0.6923720240592957, 0.22786490619182587, 1.287198543548584, 1.0238981246948242, 1.5731620788574219, 0.4158324897289276, -0.7373071312904358, 1.135249137878418, 2.092482089996338, -1.1997102499008179, -1.5658835172653198, -1.0709660053253174, 0.958372950553894, 0.7204482555389404, -0.00989673100411892, -1.450293779373169, 1.98567533493042, 1.2025644779205322, -1.5459256172180176, 0.39438313245773315, 0.548976480960846, -0.326389878988266, 2.4089229106903076, 0.1661987602710724, 0.9360279440879822, -1.5987907648086548, -0.8966164588928223, 1.1342183351516724, -0.15249942243099213, -0.14068640768527985, -0.23548609018325806, -0.5395066738128662, -0.8956859111785889, 1.8010107278823853, 0.8763008117675781, -0.12016860395669937, 0.1698109656572342, 1.6421878337860107, -0.01661519519984722, 0.5850223898887634, -0.19994094967842102, 0.18303491175174713, 1.3058674335479736, -0.668789267539978, 0.41799670457839966, -0.7758944630622864, 1.2240309715270996, -0.8420310616493225, 1.6842809915542603, 0.7783055901527405, -0.10311286896467209, -0.1690029352903366, -0.25670138001441956, 2.5207955837249756, -0.4491100311279297, -0.012037915177643299, -0.1995219886302948, 0.3799608647823334, -1.8986947536468506, 2.0756683349609375, -0.8519688844680786, 1.4886207580566406, 0.42754241824150085, 2.776662826538086, -1.1348577737808228, 0.0010333812097087502, -0.4867076575756073, 0.05140466243028641, 1.681850790977478, 0.36806246638298035, -0.14497625827789307, 0.7758438587188721, 0.7323654890060425, -0.30257704854011536, -0.39990514516830444, -0.19739463925361633, -1.2293503284454346, -1.5779480934143066, -1.6028382778167725, 2.5067954063415527, 0.8188193440437317, -0.343467652797699, 1.7656611204147339, -1.1034696102142334, 1.1165494918823242, 0.35003483295440674, 0.6706831455230713, -0.23803280293941498, 0.7272611260414124, 0.38335785269737244, 1.3006958961486816, -0.5137254595756531, 0.5611725449562073, 1.8492923974990845, 1.2945728302001953, -1.4389030933380127, -0.21882763504981995, 0.3510158658027649, -0.9790267944335938, -1.1804635524749756, 0.48582616448402405, -0.43075308203697205, -0.35057103633880615, 0.06491247564554214, 0.5049639344215393, 1.3293293714523315, -0.711326539516449, -1.1545027494430542, -1.748699426651001, -1.1525572538375854, -0.9535326361656189, -0.15631863474845886, 1.1067790985107422, 0.5887609124183655, -0.5667181611061096, -0.26584529876708984, -0.2262454777956009, 0.2584899663925171, 0.8373843431472778, 0.4667258560657501, 0.35118958353996277, -0.13191954791545868, -0.9218001961708069, -0.45458558201789856, 1.177180290222168, 1.1968938112258911, 0.021217359229922295, 0.7527886629104614, -1.6960647106170654, 2.6132822036743164, -0.21291649341583252, -1.0864726305007935, -1.257095217704773, 0.37537267804145813, 1.5727490186691284, -0.2343517690896988, -0.34001561999320984, -1.5719952583312988, 1.1581380367279053, 2.4367523193359375, -1.1853013038635254, 1.0129245519638062, 2.501351833343506, -1.039634108543396, -0.6261903643608093, -0.8060347437858582, 0.95562344789505, 1.1648356914520264, -0.7081519365310669, 1.9373408555984497, -0.1420583873987198, -1.7053439617156982, -1.7009400129318237, 0.1294977217912674, 1.2621047496795654, 0.003249012166634202, 0.21548883616924286, 1.0446240901947021, 0.587814450263977, -0.9699946641921997, -0.9359157085418701, -1.3116610050201416, 0.5719245672225952, 0.22377634048461914, 0.9626774191856384, -1.0360468626022339, 0.9387863874435425, -0.48471397161483765, 0.37646323442459106, 0.10961131751537323, -1.076727032661438, 0.5554583668708801, -0.40241339802742004, -0.10235551744699478, -1.3900583982467651, -0.5728961825370789, 1.241736650466919, -0.872194766998291, 0.7295054197311401, -1.1162585020065308, -0.49083611369132996, 0.9792510867118835, 2.065361499786377, 1.2850346565246582, -2.0401275157928467, -1.452065348625183, 1.4553025960922241, 1.568602442741394, -0.04653177037835121, -0.961576521396637, -1.9162571430206299, -1.2240824699401855, 0.03291535750031471, 0.23941196501255035, -0.9150975346565247, -0.10090259462594986, 2.0992469787597656, 0.939754068851471, -1.1903961896896362, -1.5882130861282349, -2.4165427684783936, 1.4744938611984253, -0.7222487926483154, -1.957377552986145, -0.15662653744220734, 0.23552918434143066, -1.0701258182525635, 0.6626284718513489, -0.9411734342575073, -0.6994566917419434, 0.5929569005966187, 0.12848317623138428, -1.739649772644043, 0.2897823750972748, 1.3711988925933838, -0.6403390169143677, -0.5879050493240356, -1.4269018173217773, -0.43690115213394165, 1.9337671995162964, 0.966935932636261, -0.5617562532424927, -1.0652142763137817, -0.5021926164627075, -0.3567564785480499, -0.9763166904449463, -1.993265151977539, 0.45244598388671875, 1.1671960353851318, 1.4386991262435913, -0.2895122766494751, 1.2305527925491333, 0.9841084480285645, -0.6040809154510498, 1.35944402217865, 0.09685312956571579, -0.3672383725643158, 1.5949079990386963, -0.06452810019254684, 0.19380852580070496, 2.1318066120147705, -1.4403935670852661, 0.2423921525478363, -1.6250710487365723, -0.36892786622047424, -0.3555094301700592, -1.0909910202026367, 0.337462842464447, -1.0886725187301636, 0.16739137470722198, -0.7867192625999451, 2.0955469608306885, -0.6305662989616394, 0.9918941855430603, -1.2281290292739868, -0.001912781735882163, -0.8405742049217224, -0.9694876074790955, 0.682964563369751, 0.7076911330223083, -1.6924209594726562, -0.6582970023155212, 0.7777471542358398, 0.858958899974823, 0.5165363550186157, 2.481548309326172, -0.8143175840377808, 0.92545086145401, -0.2821962237358093, -0.7979426383972168, -0.7075299620628357, 0.5630245804786682, -0.4503629803657532, 0.09677023440599442, -0.21629966795444489, -0.7315927147865295, -0.42094293236732483, 0.6017530560493469, -0.1358049213886261, 0.5246956944465637, 0.13004998862743378, 2.1975533962249756, 1.1218287944793701, 0.3618432283401489, -0.9727028012275696, 0.08533167839050293, 0.2959982454776764, 0.8161739706993103, -0.5561458468437195, 0.9714396595954895, -1.245867133140564, -1.3015955686569214, 0.7640771865844727, -0.2326093316078186, -0.6805035471916199, -0.989571750164032, 0.11047861725091934, -1.2979027032852173, -1.8510037660598755, 0.22764238715171814, 0.6969423294067383, -0.4657663106918335, 0.8389797806739807, 0.9157423377037048, -1.0840238332748413, 0.760951817035675, -0.032116517424583435, 1.971490740776062, 1.3307650089263916, -1.1518077850341797, -0.09726617485284805, -0.37693309783935547, 1.011408805847168, 1.2251005172729492, 0.2467740923166275, 1.6859817504882812, -0.5005557537078857, -1.4180222749710083, -1.194793701171875, -0.27597513794898987, -0.7101384401321411, 0.020904652774333954, 0.5927963852882385, -1.0749765634536743, 0.08893871307373047, 0.4582255780696869, 0.1640947312116623, 0.04802447557449341, -0.5585716366767883, -0.005828481633216143, 2.8570775985717773, -0.7113252878189087, -1.009644627571106, -1.9838842153549194, -0.5216600298881531, -1.1294323205947876, -1.3046952486038208, -0.7403002381324768, 0.04020475223660469, -1.2544525861740112, -0.6868312954902649, 0.4681553244590759, -0.6401088237762451, -0.23892274498939514, 1.0152581930160522, -1.3035390377044678, 0.8731414675712585, 0.46417057514190674, -0.2335941642522812, -0.5065677165985107, -1.1204557418823242, -0.840097188949585, -1.3269821405410767, 1.5116716623306274, -0.9749930500984192, 0.573969841003418, 0.2985938489437103, 0.5582903027534485, -0.9749181866645813, 1.6679915189743042, 0.29209646582603455, 1.159848690032959, -0.33745792508125305, 1.017265796661377, 0.20801027119159698, 1.0228945016860962, -0.9994834661483765, -0.6446447372436523, -0.7201525568962097, 0.19423764944076538, -1.2126010656356812, 0.9349944591522217, -0.49196818470954895, -1.1907892227172852, 0.7663004994392395, -0.0704815685749054, -0.9799473881721497, 0.12475772947072983, 1.6704587936401367, -1.3860595226287842, 0.749285101890564, -0.8974336385726929, -1.6023290157318115, 0.2870151102542877, 0.7926393747329712, 0.10867060720920563, -1.0634690523147583, -0.3256829082965851, -0.9648904800415039, 0.21043740212917328, -1.9566627740859985, -0.09962420910596848, -1.054481029510498, 1.1322308778762817, -1.7686694860458374, -0.27718988060951233, -0.6424738764762878, -0.6014246344566345, -1.0518114566802979, -1.2979052066802979, -0.833486795425415, 0.3821127414703369, -0.8256551623344421, -1.521296739578247, -0.4198341369628906, -1.3285324573516846, -0.6979373097419739, 0.9526200294494629, -1.111811876296997, 0.008336147293448448, 0.5228853225708008, 0.18266965448856354, -1.926200270652771, 0.927692174911499, -0.9217888712882996, -1.1102650165557861, -1.5108528137207031, 0.6395120024681091, -0.9633427858352661, 0.3059428930282593, 0.3599155843257904, -0.40729570388793945, -0.985802948474884, -0.11776319891214371, 0.4563741683959961, 2.2769412994384766, 2.7485098838806152, -1.7279328107833862, -0.10823113471269608, -1.1118353605270386, -0.0749044120311737, -0.239558607339859, -0.579007625579834, -0.428846538066864, 0.10120616108179092, 0.13071882724761963, 0.41258108615875244, 0.13632206618785858, 1.3741487264633179, -1.7563238143920898, -1.0235402584075928, -1.2260600328445435, -1.4379496574401855, 0.02963659167289734, -0.4037253260612488, 0.5371579527854919, -1.5482704639434814, -2.622821092605591, 0.15521685779094696, -0.9922094345092773, 0.020881323143839836, 0.5985985398292542, -1.671531319618225, -0.11912117898464203, 0.11344844847917557, 1.086475133895874, 0.7470679879188538, -0.1603195071220398, -0.39887183904647827, -1.9580185413360596, -0.6237688064575195, 0.6301101446151733, 2.085557222366333, -0.19818353652954102, 0.0838981494307518, -0.27594321966171265, 0.9550703167915344, 1.5726715326309204, -0.28526532649993896, 1.50100839138031, 0.5339269042015076, -0.8580788969993591, -1.8430087566375732, -0.5874560475349426, -0.7504934072494507, 0.7968403100967407, 1.7567042112350464, -0.4317346215248108, 0.6140491962432861, 1.2153537273406982, -0.19109973311424255, 0.1266457438468933, 0.5465183854103088, 0.8028615117073059, -0.436922550201416, -0.8888179063796997, 0.19465936720371246, -0.19288264214992523, 1.373152732849121, 0.49907395243644714, -1.6865202188491821, -0.9411804676055908, -0.3861234486103058, 0.27160772681236267, 0.7465589642524719, -1.0798389911651611, 0.17134207487106323, -1.3428218364715576, 0.10203047096729279, 0.2514585852622986, -0.9938040375709534, 0.7986049652099609, 1.3570727109909058, -0.666246235370636, -1.3840365409851074, -0.825888454914093, -0.8010976910591125, -0.9530344009399414, -0.6150549650192261, -1.5622937679290771, 1.2725310325622559, -0.6695037484169006, -0.13069991767406464, 0.6674460172653198, 0.26790139079093933, -0.14444710314273834, 1.3658021688461304, 0.6272749900817871, 1.2469993829727173, 1.3694233894348145, -1.1174147129058838, -1.3568350076675415, 0.7559648156166077, -0.4895642399787903, -0.06018543243408203, 0.07597105205059052, 0.3738754391670227, -1.4739314317703247, -0.06834795325994492, -0.5044532418251038, -0.7355424165725708, -0.012587280943989754, -0.7341719269752502, -2.1295695304870605, -0.5837580561637878, 0.3268650770187378, 1.3685837984085083, 0.20924419164657593, 1.3566925525665283, -0.2150476723909378, 0.4290519058704376, -0.45619848370552063, 0.10241065174341202, 0.32358089089393616, -0.219418004155159, 1.0177102088928223, -2.272667646408081, -0.645645260810852, 2.4005966186523438, 1.0326886177062988, 0.1747485101222992, -2.2922561168670654, -0.7750643491744995, -0.1852405071258545, -1.3594286441802979, 0.04187172278761864, 0.5672820210456848, -0.45995503664016724, 0.5580708980560303, 0.13101133704185486, -1.232962965965271, 0.5655749440193176, 1.000183343887329, -0.3969151973724365, 0.5804104804992676, 0.2678362727165222, -1.3933228254318237, 1.6560081243515015, 0.708800196647644, -1.8188520669937134, -1.5109144449234009, -0.3117232322692871, -0.4153117537498474, 0.1769217699766159, -0.08331426978111267, -0.7066524624824524, 0.4630633294582367, 0.02615833282470703, 1.1830374002456665, -0.501310408115387, -0.42334800958633423, 1.3406431674957275, 2.3904833793640137, 1.1769403219223022, -1.0000687837600708, -0.6883490085601807, 0.7660089135169983, -0.9928570985794067, 2.5518076419830322, 0.3403307795524597, 0.7354828119277954, -1.020422101020813, 0.3994913101196289, 0.6150177717208862, 0.30971771478652954, -0.3966808021068573, -0.5284338593482971, -0.7685510516166687, 0.8769056797027588, 0.5603482723236084, 0.5262026190757751, -1.4759929180145264, 0.46519723534584045, -0.7746595144271851, 0.6735155582427979, -1.9259861707687378, 0.4381902813911438, 0.47165197134017944, 0.5006105899810791, 0.846652626991272, -0.29416701197624207, -0.3902321755886078, -0.5720826387405396, 0.38192325830459595, 1.0697388648986816, 0.6237437725067139, -1.2958850860595703, 0.05182652920484543, -0.7841864228248596, 0.5895086526870728, 0.33582645654678345, -0.4514527916908264, -0.2602629065513611, 0.3220631182193756, 0.8534293174743652, 0.4046546518802643, 0.09103429317474365, 0.7763127684593201, -0.4071792662143707, 1.560238003730774, 1.0424556732177734, -0.48526206612586975, 1.2142488956451416, 0.050126124173402786, -1.3335353136062622, 0.4667952358722687, 0.06934870034456253, -0.7267605662345886, -2.0228078365325928, -0.7259815335273743, -1.4080400466918945, 0.04640776664018631, -0.5526131391525269, -0.5635067224502563, 1.5348296165466309, -1.92787766456604, 0.339339941740036, -0.6842271685600281, -0.08411502838134766, -0.5506766438484192, 1.6707184314727783, 0.17528533935546875, 1.0557246208190918, -1.4534939527511597, 1.262563705444336, -0.033746495842933655, 0.4655427038669586, -0.5722329616546631, -0.6699232459068298, 0.32937148213386536, 2.1888792514801025, 0.459151953458786, 0.9170050621032715, 0.5865932703018188, -0.44589483737945557, 0.09506477415561676, 0.3611924648284912, -0.7119459509849548, 0.696498453617096, 0.7849327325820923, -0.8267194032669067, 0.0671868696808815, -0.6498609781265259, 0.1356961876153946, -0.9731530547142029, 3.640012264251709, -0.9718806743621826, -1.6438815593719482, -0.42306065559387207, -0.8366768956184387, -0.1758902221918106, -0.40451934933662415, 1.8951988220214844, 0.3758387863636017, -0.6962218284606934, 1.6598565578460693, 0.5684396028518677, -1.2251304388046265, -0.00043769855983555317, -0.7905585765838623, 1.011283278465271, -0.9141935110092163, -1.1370129585266113, 1.0213559865951538, 0.619367778301239, 1.357602596282959, 0.11999625712633133, 0.11410234868526459, -0.6506746411323547, -0.3438766300678253, 0.5060558915138245, 1.1777864694595337, 0.11269880831241608, 2.33026123046875, 2.2940351963043213, -0.4367680251598358, -0.47293201088905334, 0.15164819359779358, 0.008784646168351173, 0.1883593499660492, 2.5049095153808594, 1.6987055540084839, 1.806947112083435, -1.2595118284225464, 1.5783025026321411, 0.7731438279151917, 0.9938013553619385, 0.40721747279167175, 0.7023167610168457, -0.6414003968238831, -0.9016494154930115, 0.4699242413043976, 0.32823988795280457, -0.27150243520736694, 0.9501569867134094, -0.7369244694709778, -0.022442670539021492, -1.0718038082122803, 1.336013913154602, -0.23670440912246704, -1.2214807271957397, -0.5861626863479614, 0.2377042919397354, 0.45321303606033325, -2.2531943321228027, -0.3146078586578369, -1.0987906455993652, -0.027366045862436295, 1.2525722980499268, 0.34659093618392944, -1.7909730672836304, 0.37411484122276306, 0.7498550415039062, 1.4745686054229736, -0.303554505109787, 0.1778680980205536, -0.558364987373352, -0.9420273303985596, 0.7507380843162537, -0.5063947439193726, -0.8659204244613647, -0.7765677571296692, 0.5013445019721985, -0.2734138071537018, 0.6156032681465149, -0.3120427131652832, 0.8666667342185974, -0.34921547770500183, -0.38255780935287476, -0.37509623169898987, 1.1325862407684326, 1.1473608016967773, -0.27139946818351746, -1.5913835763931274, -0.1660856157541275, 0.804288923740387, -0.560847818851471, -0.011852873489260674, 1.3212037086486816, -0.04028184339404106, -0.14097383618354797, -1.5964632034301758, 0.677855372428894, -0.12303610146045685, -0.3411974608898163, 0.6170300245285034, 1.1929112672805786, -0.12273042649030685, -0.4469612240791321, -0.7364916205406189, 1.0952706336975098, 1.269182562828064, 0.40076887607574463, -0.042218971997499466, 0.8517014384269714, -0.01931162178516388, 0.5596131086349487, 0.7577744722366333, -0.3909946382045746, 2.3742575645446777, -1.0433440208435059, 0.38142645359039307, -0.11245669424533844, 0.39290937781333923, 0.8352013826370239, -0.2413949966430664, 0.20525579154491425, -0.9709769487380981, 0.5377833843231201, -0.266915500164032, 2.3331148624420166, -1.0435621738433838, 1.3748321533203125, -0.6714897751808167, -0.1065477579832077, 1.6943320035934448, -0.5045088529586792, -0.28474491834640503, -0.22805267572402954, 0.8164456486701965, -0.4353334307670593, 0.4776114523410797, -0.059426527470350266, -0.047300178557634354, 0.6621688604354858, 0.8800053596496582, 1.423134207725525, 1.5635857582092285, 0.3802112340927124, -1.7417048215866089, 0.4695119857788086, 0.5635627508163452, 1.565309762954712, -0.17555661499500275, 0.43032029271125793, 0.147599995136261, 1.521541953086853, 0.38085508346557617, -0.9705770611763, -0.27444180846214294, -0.015348331071436405, -2.4237396717071533, 0.2886020243167877, 0.7394019961357117, -0.15472270548343658, 1.2179948091506958, -1.0356537103652954, -0.30739784240722656, -0.07259684056043625, 1.7956953048706055, 1.1834464073181152, -0.9077928066253662, 1.2242636680603027, 0.3597431778907776, 1.4840253591537476, 0.15909507870674133, 0.49886438250541687, 0.7072175145149231, 1.8679969310760498, -0.6029601693153381, 0.5483395457267761, -1.769675612449646, 1.1104722023010254, -0.953605055809021, 0.53722083568573, -0.7979600429534912, 0.7298780679702759, -2.2042126655578613, 0.031880103051662445, 0.07407497614622116, 0.5055357217788696, 0.029388757422566414, -0.1988663226366043, 0.06927920877933502, 0.6058764457702637, 0.20889399945735931, 0.6255717277526855, -1.5123907327651978, 0.2705830931663513, 0.8916949033737183, -1.5247353315353394, -0.6912854313850403, 0.43160519003868103, 1.003390908241272, 1.9975508451461792, -1.535668969154358, -0.26610201597213745, -1.2566591501235962, -0.6774688363075256, 0.17329780757427216, -0.4833143651485443, -1.207344889640808, 0.14201435446739197, 0.053257618099451065, -0.4264760911464691, 2.078061580657959, 1.2747859954833984, -0.6375083327293396, -0.14407116174697876, -0.23768241703510284, 0.05564955621957779, 1.491516351699829, -0.7140653133392334, -0.6507729291915894, -1.0371555089950562, -1.2532093524932861, -0.4165613055229187, 1.1892940998077393, -1.1110392808914185, -1.1744168996810913, 0.15046104788780212, 0.7116694450378418, -0.8165956139564514, -1.298904538154602, -0.46680983901023865, -0.40970996022224426, -0.1041158139705658, -1.2945979833602905, -1.7760744094848633, -0.719281792640686, 0.3045600652694702, -1.3917101621627808, 0.6531190872192383, -0.1631031185388565, -1.3439909219741821, 0.01662745326757431, -0.37873223423957825, 0.06840132176876068, 1.25899076461792, -0.8795077204704285, 0.6649026274681091, -1.8650864362716675, -0.2796884775161743, 0.9949328899383545, 0.3878540098667145, -1.406410813331604, -0.8972309827804565, -1.6441924571990967, 0.3678865134716034, -0.46352311968803406, -0.5559123754501343, 0.3040945529937744, -1.8710447549819946, 1.549243450164795, 0.21294188499450684, 0.4665140211582184, -0.2767556309700012, 1.3933179378509521, 0.46034136414527893, 0.8706310391426086, -0.8963844180107117, -0.7878608107566833, -2.1259348392486572, -0.001810609013773501, -0.4214418828487396, -0.5834142565727234, -0.5388416647911072, 0.4581833779811859, 0.5823585987091064, 1.5745480060577393, -0.356101393699646, -1.086690902709961, 1.2614134550094604, -0.10284914076328278, -0.38986721634864807, 0.14702987670898438, 1.1559064388275146, -1.3038618564605713, -0.44475752115249634, 1.5607191324234009, 0.07676126807928085, 1.7440600395202637, 1.9253615140914917, 0.5725287795066833, 0.038149550557136536, 0.9732861518859863, 0.9911529421806335, 1.433172583580017, -0.8466342687606812, 1.5409138202667236, -1.0798492431640625, 1.2240113019943237, 0.0026033809408545494, 2.8197109699249268, -0.7623417973518372, 0.2565768361091614, 0.09099498391151428, -0.06103387475013733, -0.9343032240867615, -0.023789305239915848, 0.38533836603164673, -0.49895238876342773, -0.2773750126361847, 1.3547699451446533, -0.1891753226518631, 0.14142248034477234, -0.31338000297546387, -0.9953513741493225, -0.14303988218307495, 0.4657299518585205, 1.0458515882492065, -1.474560260772705, -1.5938791036605835, 0.9754965901374817, 0.5180050134658813, 1.3176329135894775, 0.09220436960458755, -0.24219977855682373, -0.8499265909194946, 1.020804524421692, -0.48424893617630005, 1.6833813190460205, -2.119236707687378, -1.1708709001541138, 0.6087137460708618, -0.667391300201416, 1.1839871406555176, -1.1957584619522095, 0.240819051861763, -0.9596003293991089, -0.8843024969100952, 0.4775983989238739, 1.1275551319122314, 1.3794927597045898, -1.7369449138641357, 0.5768827199935913, 1.2113827466964722, -0.5257394909858704, -0.27570876479148865, 1.303847312927246, 0.932670533657074, 1.913201928138733, 1.2750685214996338, 0.42520323395729065, 0.20900148153305054, -0.24545517563819885, -0.7759989500045776, -0.9897461533546448, -0.5292655825614929, -0.6320606470108032, -0.25936198234558105, 0.15989354252815247, 1.1946278810501099, -0.1434888392686844, -0.4107934534549713, 0.32158756256103516, -0.6322324275970459, 0.27114614844322205, 0.3906930685043335, 1.094664454460144, -0.15469008684158325, -0.47842276096343994, 0.5817312002182007, -1.2731187343597412, -0.23252588510513306, 0.29121342301368713, -1.1026341915130615, 0.2772250175476074, 0.9397506713867188, -0.445517361164093, 1.0229114294052124, -0.8064009547233582, -1.9935144186019897, -1.6343305110931396, 0.4322621524333954, -0.03587622940540314, -1.8032952547073364, 0.9597898125648499, -0.4945216774940491, -0.6104499697685242, -0.2283245474100113, 0.9496974945068359, 0.6805552244186401, 0.9830806851387024, 0.08245394378900528, 0.5606422424316406, 0.4322394132614136, -0.9374805688858032, 1.0154144763946533, 0.017329201102256775, -0.2529756426811218, -0.45792654156684875, -0.3173314034938812, -1.3350697755813599, 0.37723878026008606, -1.9151231050491333, 1.249668836593628, -0.46243372559547424, -0.4105257987976074, 0.7573995590209961, -0.3038083612918854, -1.09745454788208, 1.5601452589035034, -0.5058937668800354, 1.3728556632995605, 2.60099458694458, -0.7056497931480408, -0.23117676377296448, 0.8765802979469299, 0.9782852530479431, -1.1947332620620728, 1.8397860527038574, 1.6701278686523438, 0.13386468589305878, 1.9593719244003296, -1.646684169769287, -1.5082224607467651, 0.1742209792137146, 1.2385727167129517, -1.5193191766738892, 1.259618878364563, -0.44638168811798096, 0.005511384457349777, 0.18040668964385986, 0.2506435811519623, 0.5577377676963806, 1.7468570470809937, 1.388832688331604, -1.1598409414291382, 1.0077483654022217, 0.16862818598747253, -0.23513607680797577, -1.7568306922912598, -1.8225057125091553, -0.6631976962089539, 0.43252164125442505, -0.23620808124542236, 0.6115500926971436, -1.538277506828308, -0.5082193613052368, -1.2581896781921387, 0.7618682384490967, 0.3285408020019531, -1.3734462261199951, 2.143289089202881, -0.7119092345237732, -1.4527186155319214, -0.6761154532432556, 0.2571590542793274, -0.1849418580532074, 0.23388683795928955, -0.3352239429950714, 0.5429723262786865, -0.25454291701316833, 1.271978497505188, 0.6943516731262207, -0.03246907889842987, 0.6609381437301636, 0.8009848594665527, -2.040632486343384, -0.27687495946884155, -0.18087632954120636, -0.06526438891887665, -0.02261010929942131, -0.6949559450149536, -1.1660653352737427, 0.02014804072678089, 0.18246549367904663, -0.0033530069049447775, 0.3539040684700012, -1.1432427167892456, 1.8511008024215698, -0.03388284146785736, -0.4864218235015869, 0.17413337528705597, 1.1516308784484863, 0.893550455570221, 0.6088008880615234, -0.2764946222305298, -1.344161868095398, -1.2485716342926025, 0.37342599034309387, 1.462152123451233, 1.261383056640625, -0.25421538949012756, -1.4884555339813232, -2.1394033432006836, -0.4435840845108032, -0.8904601335525513, -0.07898424565792084, 0.1924259215593338, -1.8416812419891357, 1.7507002353668213, 2.413165330886841, -0.45214828848838806, -0.2059430629014969, 0.4108000099658966, -0.38013342022895813, -0.3994632959365845, -0.295820027589798, 0.9500340223312378, 0.4895347058773041, 0.5790584683418274, -0.6918282508850098, -1.1458371877670288, 2.9135162830352783, 0.22670310735702515, -0.22853153944015503, -1.31248939037323, -1.0168416500091553, 0.6029366254806519, -0.01117404829710722, -2.0626533031463623, 0.4774799942970276, -0.4981360137462616, -0.6507518887519836, -0.5037652254104614, -0.4653906226158142, 0.18062736093997955, -0.3163856565952301, -0.8541253209114075, -0.7586408853530884, -0.12173270434141159, -0.27938106656074524, 0.9534706473350525, 0.06868155300617218, 1.1134401559829712, 1.2535200119018555, -1.36233651638031, -0.6657348275184631, 0.0685228481888771, -0.4953591227531433, -0.2983677387237549, 1.8187150955200195, 0.84871906042099, -1.2223018407821655, 0.24005450308322906, 1.7214031219482422, 0.36891427636146545, -0.7873938083648682, -1.3951209783554077, -1.7371317148208618, -1.901149034500122, 0.27925193309783936, 1.1508557796478271, 0.5206451416015625, -2.685316562652588, -1.4068200588226318, 2.0487797260284424, -0.31467369198799133, -0.6850497722625732, 0.3932907283306122, -0.838458240032196, -0.19479988515377045, 0.7715827226638794, -0.7976968884468079, -2.29378342628479, 0.26222383975982666, -0.18092766404151917, 0.3106643855571747, -1.0570805072784424, 2.17270565032959, -0.42055585980415344, -0.4090828001499176, 0.5144853591918945, -0.09220346063375473, 0.8409239649772644, 0.5963867902755737, -0.5540093779563904, -1.209683895111084, 0.6325421929359436, -0.499651700258255, 1.1998158693313599, 2.0024592876434326, 1.2873731851577759, -0.34542912244796753, 1.3310774564743042, -0.63752681016922, -0.6074292063713074, 0.34643346071243286, -0.22877652943134308, -0.26971137523651123, -1.5529042482376099, 0.2588889002799988, -1.3017667531967163, 1.3319095373153687, 0.8556897640228271, -0.12641143798828125, -1.5239182710647583, 0.2058687061071396, 1.8008297681808472, 0.7814289331436157, 1.5842293500900269, 0.51017165184021, -0.27065524458885193, 0.03007209114730358, 0.9942824840545654, 2.7424116134643555, -0.5712366104125977, 2.859191656112671, -1.2845041751861572, -1.7918510437011719, -0.8773156404495239, 1.1351717710494995, -0.43350347876548767, 0.5485938191413879, 0.3692389130592346, -1.4344921112060547, -1.210176706314087, 1.0863335132598877, 1.0848534107208252, 1.4889675378799438, -0.663033664226532, -0.8074922561645508, -0.37115925550460815, -2.4457662105560303, -0.06180796027183533, 0.9433236718177795, -0.8207828402519226, 0.7152342200279236, -0.06090924143791199, 1.2475543022155762, -1.2922698259353638, 1.2083137035369873, 1.1801247596740723, 0.6203781366348267, -0.5997412800788879, 0.30784526467323303, -1.4621970653533936, -0.07027245312929153, 1.0920076370239258, -1.8877094984054565, 0.6714289784431458, -0.221905916929245, 0.1881333887577057, -1.5307762622833252, -0.21681088209152222, 0.3260308802127838, -1.6030069589614868, 0.10093662887811661, -0.08603135496377945, -0.6203663945198059, -3.16088604927063, 0.6981764435768127, 0.012494655326008797, -0.3423853814601898, -0.027383215725421906, -1.366347312927246, 0.016583163291215897, 0.483447790145874, 0.602997899055481, -1.3289523124694824, -0.13240735232830048, 0.0035838126204907894, -0.5534303188323975, 0.5470969080924988, 0.6256980895996094, -0.6716078519821167, 0.6494677066802979, 0.029333356767892838, 1.1023441553115845, -1.0828038454055786, -0.0654621496796608, -0.08313976228237152, 0.9285763502120972, 0.4572090804576874, 1.0807044506072998, -0.4241449236869812, -0.1928333044052124, 2.776664972305298, 1.6933379173278809, 0.19782240688800812, 2.680743932723999, 0.20416142046451569, -0.5534151196479797, 1.7752423286437988, -0.22744356095790863, 0.3210166096687317, 0.0025754140224307775, -0.5186838507652283, 0.517332136631012, -1.5844858884811401, 1.8242815732955933, -0.20606189966201782, -0.18521368503570557, 0.5745282769203186, 0.874958336353302, -0.8610617518424988, 0.10436981916427612, 0.5946172475814819, 0.20210924744606018, -0.577656626701355, 1.467596411705017, 0.6043155193328857, 0.2624417245388031, 0.09028791636228561, 0.38533735275268555, -1.1243951320648193, -0.7027949690818787, -1.0869544744491577, -1.6855134963989258, -0.5907794237136841, 0.5586552619934082, -0.2969200909137726, 0.1954435110092163, 1.2331606149673462, 1.1995952129364014, 0.020838487893342972, 0.0792723298072815, -0.04837634414434433, -0.08028514683246613, 2.502687692642212, -0.8288423418998718, 1.5511237382888794, -1.0692417621612549, 0.31426823139190674, -0.8091357350349426, 1.8724572658538818, -0.8946174383163452, -0.3033958971500397, -0.22004932165145874, -0.9357218742370605, 0.5973391532897949, -0.11764734238386154, 1.1782335042953491, -0.369374543428421, 0.2814478278160095, 0.964221179485321, -0.8234162926673889, 1.2792576551437378, -0.07631585747003555, 0.18791170418262482, -0.28858786821365356, -0.1349557340145111, -0.43595388531684875, 1.1531397104263306, -0.4410403072834015, -0.1210554987192154, 0.16435754299163818, -1.103270173072815, 0.11608806252479553, -0.10132506489753723, -0.44656604528427124, 0.4559701979160309, -0.8686179518699646, 0.061084724962711334, -0.17362827062606812, 0.37435150146484375, 0.26548251509666443, -0.8898806571960449, 1.5845766067504883, 0.37807419896125793, -0.5680215358734131, 0.09753935784101486, 1.9606798887252808, 1.4851362705230713, 1.4985458850860596, -0.2812437117099762, -1.0999947786331177, 0.3566926419734955, -0.8008978366851807, 0.06620161235332489, -0.2886032164096832, 0.0063397386111319065, -0.2394922822713852, -0.05475197359919548, 0.505384624004364, 0.37102875113487244, -0.6491705775260925, 0.6338870525360107, 0.21920840442180634, -1.4085392951965332, 0.30110183358192444, -0.6742554306983948, 0.6342977285385132, -0.6282529234886169, -0.7449384927749634, -0.03806713595986366, -0.47996142506599426, 1.3561275005340576, -0.40497520565986633, 0.22774305939674377, -0.734653651714325, 2.2827165126800537, -1.2034169435501099, 0.05347545072436333, -0.10197672992944717, -0.15667809545993805, -0.7632688283920288, 1.8636016845703125, 0.6764683127403259, 0.07116392254829407, 0.7277584671974182, 0.09739474952220917, -0.7628280520439148, -2.571587324142456, -0.4485107362270355, 0.17922933399677277, 0.2504706382751465, -0.7393057942390442, 1.2985552549362183, -0.21710675954818726, 1.2132289409637451, -0.1500559002161026, 0.0020945658907294273, -0.03326747938990593, 0.519087016582489, 0.3228900730609894, -0.8593879342079163, 0.6617794036865234, 1.2225089073181152, -0.7480682730674744, -1.2308911085128784, 0.05994143709540367, 0.4353022873401642, -0.3204166293144226, -0.7269572615623474, -0.36848706007003784, 0.0883791521191597, -0.18389421701431274, -1.4922178983688354, 0.1931765079498291, -0.6982124447822571, 1.8924260139465332, -1.7480679750442505, -2.2868845462799072, -0.6223365664482117, 1.3402942419052124, 0.8754952549934387, -0.9783267378807068, 0.07755343616008759, 0.5420100688934326, 0.7319329977035522, -0.44508233666419983, -0.9134066700935364, -1.8025552034378052, 0.4472799599170685, 0.4188874363899231, 1.058712124824524, -1.0833861827850342, 0.5248875617980957, -0.7160686254501343, -0.3344080150127411, 1.2989094257354736, 0.2550196647644043, -0.3868897557258606, 1.126731276512146, 0.14549550414085388, -2.0685741901397705, 0.3478788435459137, 1.6825631856918335, -0.44591081142425537, -0.0319126658141613, -0.4475563168525696, 0.801369845867157, 0.4144524335861206, 1.2844569683074951, 1.395151138305664, -0.7613328695297241, -1.1779662370681763, 0.6947259902954102, -2.768852710723877, -0.48805034160614014, 1.2635314464569092, 0.5180907249450684, -0.720120906829834, -0.27421262860298157, -0.21342113614082336, -1.756961703300476, 1.343727946281433, 1.1847724914550781, 1.3278096914291382, 1.0448812246322632, -0.4250622093677521, -0.6977298855781555, 0.9332830905914307, 2.1497130393981934, -0.8549826145172119, 0.6152648329734802, 0.7602731585502625, 0.2477065920829773, -0.08199057728052139, -1.2924120426177979, -1.3788076639175415, 1.2082206010818481, 0.507290780544281, -0.6594592332839966, -1.5650886297225952, -0.9265755414962769, -0.6766035556793213, -0.23079538345336914, -0.2931947410106659, -0.9449940323829651, 0.2789227068424225, 1.0691965818405151, 1.1538763046264648, -0.07563107460737228, -0.5047755837440491, 0.1274086982011795, 0.016931135207414627, 0.906836211681366, 0.3112947940826416, 2.0652964115142822, -0.3837874233722687, -0.4590112864971161, 0.39022091031074524, 1.8551557064056396, -0.29160556197166443, 0.2588514983654022, 0.36810460686683655, -0.010456442832946777, -0.1909642517566681, -2.3711838722229004, -1.9517979621887207, 1.1296409368515015, 1.0161722898483276, 0.11357589811086655, -1.90291428565979, -2.001777172088623, -0.509829580783844, 0.1651730239391327, -1.6004917621612549, 0.9022728800773621, -2.0831871032714844, 0.22013244032859802, -0.7158005237579346, 0.05882538482546806, -0.16757917404174805, -0.2892269492149353, 0.8302673697471619, -0.5129666328430176, 1.3133867979049683, 0.7050530910491943, -1.1196202039718628, 2.4496591091156006, -0.8023391962051392, 0.20626749098300934, 0.4938201606273651, 0.834542989730835, -0.9471918344497681, -1.1337300539016724, -0.13396631181240082, -0.0264437273144722, -0.4760015308856964, 1.2640405893325806, -1.5690914392471313, 0.2170172929763794, -0.903438925743103, -0.7353507280349731, -0.05699100345373154, 1.093057632446289, -0.7448849678039551, 0.6489841938018799, -0.798150360584259, -1.8203315734863281, 1.0001940727233887, 1.4095224142074585, 2.549175500869751, -0.3602360785007477, 0.26814350485801697, 2.0875184535980225, 0.08879119157791138, -0.07581140100955963, -0.9625719785690308, -0.38474127650260925, -2.0620603561401367, 0.8976895213127136, 0.3226030468940735, 1.043799877166748, 1.5516382455825806, -0.44260168075561523, -0.411037415266037, -0.5764626264572144, 1.085458517074585, -1.3988494873046875, 0.6269262433052063, -1.7494395971298218, -0.5607884526252747, -0.38084501028060913, -0.3720320463180542, 1.368569016456604, 0.5939151048660278, -0.38195154070854187, -0.007441701367497444, -1.0334502458572388, 0.2978605329990387, 0.2742455005645752, -1.0914570093154907, 0.06634914129972458, 0.047871433198451996, 1.041358470916748, 0.00137139274738729, 1.6611828804016113, 1.0812792778015137, 0.3787617087364197, 0.5474417805671692, 2.5975983142852783, -1.7419238090515137, 0.254206120967865, 0.6928125023841858, -0.9663689136505127, -0.9953497648239136, -0.5676820874214172, -0.5955091118812561, 0.25969812273979187, -1.3332229852676392, 0.2924906313419342, -0.004866615403443575, -0.3887077867984772, -0.02840179204940796, -0.5266879796981812, -0.010778995230793953, -1.532274842262268, 0.14294232428073883, -0.6128706932067871, 1.2366787195205688, 0.018984390422701836, 0.012721621431410313, -1.0929769277572632, -1.0819759368896484, 1.2893147468566895, -2.1173477172851562, -1.3326395750045776, 0.2163192331790924, 0.08915290981531143, -0.7387261986732483, 0.708424985408783, 0.3030956983566284, 0.9689813256263733, 1.1454474925994873, -0.6443203687667847, -0.727840006351471, 0.9825707674026489, 0.6210295557975769, 0.5951130390167236, -0.3320356011390686, -1.2296276092529297, -2.029569149017334, -0.385684072971344, 1.2084393501281738, -0.06865614652633667, -0.4310854375362396, 0.7896370887756348, 0.5555201768875122, 0.09971784800291061, 0.8338453769683838, 1.623700499534607, -1.1338032484054565, 0.2735333740711212, 0.6642089486122131, -0.7170248031616211, -0.3992299437522888, 1.7767012119293213, 0.22293737530708313, 0.2584441304206848, -1.147850513458252, -1.1459640264511108, 0.9251961708068848, 0.2002912014722824, 0.18276546895503998, 0.19333042204380035, -0.6512035131454468, -0.3348456919193268, 0.05323700234293938, 0.5591347217559814, -0.26309680938720703, -1.1369469165802002, 0.34392210841178894, -0.36626869440078735, -1.1811964511871338, -0.604424774646759, -0.14962933957576752, -2.250948190689087, 1.4650498628616333, 1.0042246580123901, -0.3081204295158386, -0.3482755720615387, -0.8351917266845703, 0.550808846950531, 0.08780979365110397, 0.35783636569976807, -0.2694678008556366, 0.8951404690742493, -0.9050922393798828, -0.040766410529613495, 0.01998659409582615, -0.8072460293769836]], "dtype": "float32"}, {"__nd__": [0.5767450332641602, 1.5927067995071411, -0.37729203701019287, -0.9781373143196106, 0.5458099842071533, -1.7969231605529785, -0.027005674317479134, 1.0551660060882568, 0.9323434233665466, -2.3222339153289795, 1.6448084115982056, 0.7506152391433716, 0.5533108711242676, 0.20714856684207916, 0.3445250689983368, 0.8837210536003113, -0.5850294232368469, 0.23037417232990265, 0.15663985908031464, 0.803235650062561, 1.4516844749450684, 0.36028870940208435, -0.04874734207987785, 1.2332104444503784, -0.1931157112121582, -0.726783275604248, 0.4638090133666992, -0.46207746863365173, -1.0529993772506714, 0.06206965819001198, 1.0319703817367554, 0.6871456503868103, -0.420227974653244, -0.4987255036830902, -0.08743196725845337, 1.4867477416992188, 0.4734687805175781, -0.1710614562034607, 0.6422210335731506, -1.0010395050048828, -0.8994077444076538, 0.9853976368904114, -0.17798012495040894, 0.2580725848674774, 0.32149749994277954, -0.7321996092796326, 0.19173717498779297, 1.0592756271362305, -0.9811298847198486, -0.27297019958496094, 0.9565866589546204, -0.33167126774787903, 0.13014894723892212, 0.8414945602416992, 0.09271711111068726, -0.5901033282279968, 0.2039569765329361, -0.005113763269037008, 0.056931138038635254, 0.055168043822050095, -0.5242123007774353, 0.2212807983160019, 0.29877546429634094, -0.7150772213935852, -1.4079843759536743, -0.1936500370502472, -0.6241599917411804, 1.824242353439331, 0.1733366847038269, 0.7423948645591736, -0.13247936964035034, -1.4697420597076416, -0.6435667872428894, -1.6179808378219604, 0.7632664442062378, -0.43211641907691956, 0.30125692486763, 2.2717764377593994, 0.0451100654900074, 0.47956588864326477, 1.9192593097686768, 0.654346227645874, 0.1665874719619751, 0.12612996995449066, -0.34336888790130615, -0.2812298536300659, 0.17363432049751282, -0.8079758286476135, 1.068293571472168, -1.1601581573486328, 1.0563052892684937, 0.25907203555107117, -0.15857228636741638, -0.6085914373397827, 0.864898681640625, 1.5595786571502686, -0.20394963026046753, -0.7259882688522339, -0.8158180713653564, 0.1641131192445755, -0.07667411118745804, 0.7555630803108215, 0.7603852152824402, 0.15885429084300995, 0.18820832669734955, 0.02095112018287182, -0.24318991601467133, -0.15603333711624146, 1.7129099369049072, -0.7611191272735596, -0.9683645963668823, 0.029691534116864204, 0.36896929144859314, -0.18741248548030853, 0.6046051383018494, 0.7560038566589355, -0.8020371794700623, 1.0048102140426636, 0.8562765121459961, -1.6312705278396606, 0.4348706603050232, 1.3141380548477173, 0.14388154447078705, 1.5214921236038208, 0.061283666640520096, -1.041006088256836, 0.9922668933868408, 0.0628313198685646, -1.8430365324020386, -0.5210541486740112, 1.4031237363815308, 1.9325487613677979, 0.11445910483598709, 0.7074920535087585, -1.2644785642623901, -0.55689537525177, 0.17753247916698456, -0.791342556476593, -1.0417245626449585, 0.5067647099494934, 0.6263229846954346, 0.10509584844112396, -0.16777074337005615, 0.17791128158569336, -1.780263066291809, -1.3415074348449707, 0.9589616060256958, 2.548375129699707, 0.023683901876211166, -0.14207613468170166, -1.047008752822876, 0.39849618077278137, -0.1578388512134552, 0.2332019954919815, 1.6754080057144165, -0.746673047542572, -1.020192265510559, 0.2701890766620636, -0.5133809447288513, 1.0682564973831177, 0.827446699142456, 0.04805528745055199, 0.22475869953632355, -1.6385600566864014, 1.2109929323196411, -0.03841574117541313, -0.7628070116043091, -0.8194049000740051, 0.4485723674297333, -0.06543561816215515, -1.250236988067627, 1.194366693496704, 1.0231698751449585, 1.5132330656051636, -0.26712167263031006, -0.16221997141838074, -0.21579426527023315, -1.217383623123169, -0.5281493663787842, -1.1482014656066895, -0.5896856784820557, -0.050483401864767075, 0.04838816449046135, -0.46282827854156494, 1.4990805387496948, -1.2637537717819214, 0.5149769186973572, -0.013292325660586357, -0.735598623752594, 2.6288442611694336, 0.2564489245414734, 1.7087477445602417, 0.38963228464126587, -1.4762065410614014, -0.6763784885406494, 0.4664539396762848, -1.7114348411560059, 0.3245890140533447, 2.1138174533843994, 1.4256632328033447, -0.27083876729011536, 0.1255270093679428, -0.8546761274337769, 0.09359115362167358, 0.4694940745830536, 0.17156757414340973, 1.5915453433990479, -0.9338663220405579, -1.7515816688537598, -0.27565187215805054, 0.12134133279323578, -0.2539580166339874, -0.1547384262084961, -0.05357835814356804, -0.24995504319667816, 1.3490772247314453, 1.1072008609771729, -0.5729066133499146, -0.09164173156023026, 0.23502375185489655, -0.8702965974807739, -1.6554358005523682, 0.8631629943847656, 1.690278172492981, 0.7998741865158081, -0.1792822629213333, 1.6746340990066528, -1.466566562652588, -2.0477094650268555, 0.09977424889802933, 0.3819409906864166, -0.9442243576049805, -1.5602836608886719, 0.8281455636024475, 1.7532823085784912, -0.7167174816131592, -0.2722383439540863, 0.47071772813796997, -0.2100946009159088, 0.04957060143351555, -1.1522973775863647, 1.0790902376174927, 1.9446791410446167, -0.6424611806869507, -0.8628968596458435, -0.17445459961891174, -2.114675283432007, 1.8783597946166992, 0.22337910532951355, 0.1666697859764099, 0.21653059124946594, -0.967158317565918, -1.3829708099365234, -0.7979569435119629, 0.9787920713424683, -0.1279350370168686, -1.5804659128189087, -0.6436313986778259, 0.03383908048272133, -1.138706088066101, 0.7144035696983337, 2.1535868644714355, -0.8565108776092529, -1.2622089385986328, -0.34290993213653564, 0.642242968082428, 1.9478250741958618, -0.4840211868286133, 1.1015247106552124, 1.607191801071167, 0.002066610846668482, 1.2281478643417358, 0.3198915719985962, -0.7026578187942505, -0.5969560742378235, 0.6451132297515869, 0.5080572962760925, -0.27411118149757385, -0.8446376323699951, -1.068898320198059, -0.11191476136445999, -1.2875410318374634, 0.8576706647872925, 0.4285137355327606, 0.6184041500091553, -0.6320573091506958, -0.9807791709899902, -1.4108154773712158, 0.33656635880470276, -1.1210415363311768, -0.18675006926059723, 0.5484127402305603, -1.6528496742248535, 2.045039415359497, 0.3849024176597595, -0.5285258889198303, 0.2683810591697693, 1.7602840662002563, -0.7728594541549683, 0.1768033504486084, 0.5769386887550354, -0.028524091467261314, 0.1723242849111557, -0.8521984815597534, -1.4050610065460205, 1.8980722427368164, -0.14433464407920837, 2.016742467880249, 0.8321250081062317, -0.571778416633606, 0.08670002967119217, -1.0464798212051392, -0.3399456739425659, -0.9597432613372803, 1.6751710176467896, 0.3531453013420105, -1.7296229600906372, 0.19815368950366974, -1.1800793409347534, -0.24891671538352966, -0.19092057645320892, -0.8246937394142151, 0.6729544401168823, -0.645983874797821, 0.1595226526260376, -0.48356449604034424, -1.5927330255508423, -2.5234484672546387, -0.5398781895637512, 1.079124093055725, -0.47930681705474854, -1.9847586154937744, -0.25932633876800537, -0.01685269922018051, -0.4908771812915802, -1.3235875368118286, -0.9994073510169983, 1.0808144807815552, -0.4297438859939575, 0.468821257352829, -0.6836516261100769, -1.5143133401870728, -1.1658248901367188, 0.9273673295974731, -0.789911687374115, 1.3154561519622803, 0.7192064523696899, 0.17268572747707367, -0.8058355450630188, 0.14566856622695923, 2.6581013202667236, -0.7297472357749939, -1.737233281135559, -1.0056147575378418, 0.4603079557418823, 0.6616557240486145, 0.36142903566360474, -0.8714184761047363, 2.949223756790161, 0.9904944896697998, -0.7328004837036133, -0.5973684191703796, -1.1152989864349365, -0.3209843635559082, -1.9379490613937378, 1.0815966129302979, -0.9052778482437134, 0.7596723437309265, -0.8943527340888977, -1.2599706649780273, 0.07594698667526245, -0.2303367406129837, -0.2392570674419403, 1.4219326972961426, 0.7848470211029053, -1.2633806467056274, 1.0945457220077515, -0.10539299249649048, 1.7455941438674927, -0.7322808504104614, -0.6172069907188416, 0.6861445307731628, -0.01931735873222351, 1.224045991897583, 1.8296493291854858, 0.33563247323036194, -0.09693106263875961, 0.09043648093938828, -1.2685736417770386, 0.7039497494697571, -0.6657208204269409, 0.07480230927467346, 0.2559869587421417, 0.08068535476922989, 0.586118757724762, 0.19608233869075775, -0.33936041593551636, 0.08111611008644104, -0.6649645566940308, -0.0002615133998915553, -0.42781123518943787, -0.7925864458084106, -0.44259655475616455, -0.7820975184440613, 0.13423985242843628, 0.9738863706588745, 1.5823897123336792, -0.42558881640434265, -1.4663385152816772, 1.8743308782577515, 0.615654468536377, -1.7602055072784424, -1.5431385040283203, -1.402603268623352, 0.04254865646362305, -0.48554590344429016, -0.29376402497291565, 0.9022203087806702, 0.4121544361114502, -0.5745101571083069, -0.7196067571640015, -1.0249749422073364, 0.6032640337944031, 1.7149139642715454, -1.4673023223876953, 1.5221151113510132, -0.026635510846972466, -1.1693649291992188, 0.32592761516571045, 0.3014391362667084, -0.4565153121948242, -0.274466872215271, -0.48525798320770264, 1.8887619972229004, 1.1422758102416992, 1.3981280326843262, 0.31164583563804626, 0.12329567223787308, -0.20234914124011993, 0.8679824471473694, 1.181471586227417, -0.4334677755832672, -0.6170279383659363, 0.12766484916210175, -1.2559573650360107, -1.1259270906448364, -1.3194833993911743, 2.0096592903137207, 0.9068261384963989, 0.010377869009971619, 1.1740849018096924, 1.5257189273834229, 0.009301821701228619, 0.35692501068115234, -1.3859331607818604, 0.47540464997291565, 0.1811995506286621, -1.104823112487793, 1.7833645343780518, -0.022779537364840508, 0.962515115737915, -0.2387925535440445, 0.05010085925459862, 0.25937989354133606, -0.16364747285842896, 0.7447178363800049, 2.366806983947754, -0.12618371844291687, -1.1722627878189087, 0.45053988695144653, -1.031404972076416, -0.9629017114639282, -0.5014632940292358, -2.26971435546875, 0.5683112144470215, -0.6802128553390503, -0.7406947016716003, -2.152094841003418, 0.5080915689468384, -1.487094759941101, -0.21577781438827515, -0.3554767370223999, -0.7277524471282959, -0.7381011843681335, 0.5515188574790955, -1.692151665687561, -0.8796961307525635, -0.5221204161643982, -1.5653630495071411, 0.004805485252290964, 1.0889145135879517, 1.3809394836425781, -0.9566972255706787, 1.2840858697891235, -0.415688693523407, 0.46445757150650024, -0.9519266486167908, -0.23718971014022827, 1.4471322298049927, -1.2510335445404053, 3.0617012977600098, -1.6492055654525757, -0.5144738554954529, 0.02642393670976162, -1.1954221725463867, 0.06856226921081543, -0.6936342716217041, 0.8753498792648315, -0.31649285554885864, 1.4590924978256226, 1.8182833194732666, 0.1503339260816574, -0.21791449189186096, 2.1153085231781006, -0.8114632964134216, 0.887583315372467, 1.9563572406768799, 0.8916158676147461, -1.3230112791061401, 1.9784613847732544, 0.6303030848503113, -0.5254115462303162, -0.39810773730278015, 0.9759001135826111, -0.696203887462616, 0.39750808477401733, 0.7047371864318848, -0.4036884307861328, -0.5345796942710876, 1.4741169214248657, -0.43269455432891846, 1.1010817289352417, 0.14976303279399872, 0.24225255846977234, 2.338958501815796, 0.8457961678504944, 1.4768848419189453, -0.5999701619148254, 2.205608367919922, 0.8446489572525024, 0.4279315173625946, 0.003943403717130423, 1.9050525426864624, -0.1998850405216217, 0.4181264638900757, 0.02530691958963871, 0.4304240345954895, 0.2349003404378891, 0.09474683552980423, -0.0365903414785862, -0.10463502258062363, 0.08988265693187714, -0.5088394284248352, 0.24241846799850464, -0.9521441459655762, -1.1708346605300903, 0.5368351340293884, 0.8004239797592163, 0.2739039659500122, 0.18534381687641144, -0.9016581773757935, -1.3301265239715576, -0.10591868311166763, -1.1046571731567383, -2.052279472351074, -1.2796586751937866, -0.2665335237979889, 0.5458608269691467, -1.7827554941177368, 0.9700586795806885, -0.38993555307388306, 1.737533450126648, -0.6505591869354248, 0.7168198823928833, 0.5015312433242798, 0.6793712377548218, 1.700386881828308, 0.4322968125343323, -1.3817991018295288, 0.6496987342834473, -1.2167742252349854, 0.9004126191139221, 0.08811481297016144, -0.16501040756702423, 0.13263724744319916, -1.5401370525360107, 0.08821722865104675, 0.30652734637260437, -0.5181856751441956, -0.12750643491744995, -0.8737974166870117, -0.2475484311580658, -0.23761437833309174, -0.11633431166410446, 0.01120017096400261, 0.03028769977390766, 1.020495891571045, 0.4390898644924164, 1.5713306665420532, 1.1446330547332764, -0.6577968001365662, -0.21471214294433594, 0.26974794268608093, 1.4501360654830933, -0.2526569664478302, -1.1918940544128418, -1.4783992767333984, 1.1391645669937134, -0.9289918541908264, -1.5268707275390625, 3.1770503520965576, -0.7014259696006775, -0.8392229080200195, 0.6626955270767212, 0.21932920813560486, -0.7022587656974792, 0.3438609540462494, 0.6436557173728943, 0.9467040300369263, 1.7075625658035278, -0.008589611388742924, -0.20112349092960358, 0.4784887135028839, 1.7552348375320435, -0.19405347108840942, -1.0326014757156372, 1.4636330604553223, 0.2552320659160614, -0.29845139384269714, -0.42585232853889465, 0.8203288316726685, 0.2345310002565384, 0.011078186333179474, 0.5155613422393799, -1.5834048986434937, -0.1609429568052292, -1.0219281911849976, 0.9077149629592896, 0.054044075310230255, 2.2033276557922363, 0.9699093699455261, -0.20455209910869598, 2.0667009353637695, 1.0938169956207275, -0.08448710292577744, 2.0436532497406006, 1.2073001861572266, 0.2439277321100235, 1.622753381729126, 0.05921907722949982, 0.8742007613182068, -1.563917636871338, 0.19284968078136444, 1.245845913887024, 0.364207923412323, -0.15970629453659058, -0.024261964485049248, 1.1729258298873901, -1.0830600261688232, 0.9554818868637085, 0.13008172810077667, -0.5231940150260925, -0.11061542481184006, -0.07650565356016159, 1.845910668373108, 0.09406289458274841, -2.561109781265259, 1.036840558052063, -0.8413823246955872, 0.07541266828775406, -0.7261099815368652, 0.2358732670545578, 0.9414505958557129, 1.8857611417770386, -1.4116785526275635, -2.505772352218628, 0.6295620799064636, -0.2399066984653473, -1.3167192935943604, -0.09032691270112991, 0.1355714499950409, 0.14179620146751404, 0.5124486684799194, 0.6424591541290283, -0.26133057475090027, -0.34131065011024475, -0.13035395741462708, 0.5815238952636719, 2.2408668994903564, -0.6335433125495911, 2.995161533355713, 0.9497714042663574, 0.3543783128261566, -0.8502188920974731, -0.8575509786605835, -1.1772724390029907, 0.2532397210597992, 0.09860530495643616, -2.134645938873291, 1.8772400617599487, 0.04696028679609299, -1.4975979328155518, -0.25627633929252625, 0.13284477591514587, -0.5444788932800293, -2.0877745151519775, 0.11538798362016678, -0.18346284329891205, 1.7872732877731323, -1.850600242614746, 0.7509773969650269, -0.8076577186584473, 1.5487293004989624, 0.5709378123283386, 0.15700171887874603, 1.687683343887329, 0.2280592918395996, -0.1526404768228531, -0.8272098302841187, -0.5197659730911255, 1.5765198469161987, 0.21561813354492188, 1.943803310394287, 1.1733742952346802, -1.4159436225891113, 0.8144140839576721, 0.09283182770013809, 0.6197530031204224, 0.361486554145813, -0.5867379307746887, -0.7837410569190979, -0.39060601592063904, 1.0449645519256592, 0.1459360420703888, -0.6483834981918335, -0.8268470764160156, -0.07788440585136414, 0.8123283982276917, -0.049668729305267334, 0.7762995362281799, -0.002712318440899253, -0.5234586596488953, -1.2841436862945557, -0.7246200442314148, -0.3629811108112335, 0.4975737929344177, -0.0013429257087409496, -0.14274679124355316, 0.595867931842804, 0.5901536345481873, 0.4157276749610901, -1.9379284381866455, -2.5631332397460938, 1.1373095512390137, -1.6389390230178833, -0.19934624433517456, -0.4263845682144165, -0.13084834814071655, -0.5112094879150391, 0.1019151583313942, -1.326889991760254, -0.06863895803689957, 0.33002084493637085, 0.4024401605129242, 0.4473345875740051, 1.5847282409667969, -0.295939564704895, 0.29229146242141724, 0.11147522181272507, -0.7854339480400085, 0.013029603287577629, 1.261684536933899, -1.0470271110534668, -0.06191602349281311, -0.009079008363187313, 0.5255480408668518, -1.1283514499664307, -0.6582798957824707, -0.5416892766952515, 1.7266550064086914, -0.02167169563472271, -1.2116605043411255, 0.22044767439365387, 0.33656951785087585, 0.6556538343429565, -0.015535233542323112, -1.5651352405548096, -0.7501357793807983, -1.2405920028686523, -1.266950249671936, 1.0718997716903687, 0.5581529140472412, -0.2986604571342468, 0.6064814925193787, -0.0027505073230713606, 0.3206264078617096, 0.13590091466903687, -0.21136920154094696, -0.294321209192276, 0.7088626623153687, 1.0978533029556274, -0.0767233595252037, -0.2961609959602356, -0.5161328911781311, -2.831542730331421, -0.6761519312858582, -0.8410330414772034, -1.2952247858047485, 0.5144568085670471, 1.7484805583953857, -0.9874358773231506, 0.06521394848823547, -1.1766606569290161, 0.13728681206703186, 0.19005697965621948, -0.1038375049829483, 0.39788326621055603, -0.07896837592124939, 1.7434300184249878, -0.7570106387138367, -0.5372861623764038, 1.2437859773635864, -1.3804534673690796, 1.0084158182144165, 3.186235189437866, -1.0022177696228027, 0.020820865407586098, -1.4362750053405762, -0.28766703605651855, -0.4579404890537262, -0.6598498225212097, 0.1334427148103714, -0.9338165521621704, -0.36204588413238525, 2.449009895324707, -1.4196500778198242, -0.622459352016449, 2.3242311477661133, 1.0412029027938843, -0.9321556687355042, -1.8106719255447388, 0.8787626624107361, 0.6906591653823853, -1.5454881191253662, -0.9824704527854919, 0.6434159874916077, 1.276534914970398, -0.6194380521774292, -0.04213588312268257, -0.21642430126667023, 0.12108708173036575, -1.3052833080291748, 0.6995261311531067, -0.16597068309783936, -0.2974521219730377, -1.2989245653152466, -0.19707809388637543, 1.060102939605713, -0.004891137592494488, -1.654049038887024, -1.0547126531600952, 1.0438367128372192, 1.8532640933990479, -0.07295157760381699, -0.34066012501716614, -0.15965673327445984, -0.47302576899528503, -0.11464780569076538, -0.6445179581642151, -0.49197742342948914, -0.5966141223907471, -0.5343571305274963, -0.9042201042175293, 0.010551471263170242, -0.27737969160079956, 1.5577702522277832, -0.028013428673148155, -1.1322100162506104, -0.5612869262695312, -0.9361488223075867, -0.41398873925209045, -0.562065064907074, -1.3108512163162231, 0.6834484934806824, -0.4307810068130493, -0.18227633833885193, 0.11000143736600876, -0.7141510248184204, 0.24314694106578827, -1.3269723653793335, -0.3370376229286194, -1.1774548292160034, -0.033553339540958405, 0.7648939490318298, -1.1428695917129517, -0.9588187336921692, 1.294921875, -0.6694405674934387, 0.5699281096458435, 1.379927158355713, -0.8415474891662598, 0.3952704668045044, 1.3006104230880737, 0.17919407784938812, 1.5202524662017822, 0.22925850749015808, -0.1714474856853485, -1.901682734489441, 0.7320346832275391, 0.3461754024028778, -0.6491158604621887, -0.6713771224021912, -0.82810378074646, 0.36385345458984375, 0.43389248847961426, -0.42665979266166687, -1.7851965427398682, 1.6620893478393555, 0.1882297843694687, 1.3042000532150269, 0.15520700812339783, 1.0582962036132812, 0.6992279887199402, -0.4858512580394745, 1.4068478345870972, 0.26230883598327637, -0.8630651831626892, 0.8266353011131287, 0.6806508302688599, 0.896561861038208, -0.4294377267360687, 0.9713060259819031, 0.31360429525375366, -1.5530149936676025, -0.2752155661582947, -1.7538312673568726, 0.33320388197898865, 1.7720741033554077, 1.2528162002563477, -1.1251879930496216, -1.1837406158447266, 0.8485579490661621, -0.4900294244289398, 1.8204421997070312, 0.3938175439834595, 0.022429808974266052, -0.238169327378273, -2.896071434020996, -0.5645374655723572, -1.6098779439926147, 0.48967650532722473, -0.5582370162010193, -0.28274115920066833, 0.30167555809020996, -0.7830982208251953, 0.3649023473262787, -1.594867467880249, 1.6919068098068237, -0.4972308874130249, -0.16639241576194763, 2.590698480606079, -1.254989504814148, 2.5169246196746826, 0.800627589225769, 1.298892617225647, -1.3660824298858643, -1.1203877925872803, 0.040049318224191666, -0.4840072691440582, -1.8809155225753784, -1.1663415431976318, 1.9383535385131836, -0.8470686674118042, 0.03769681975245476, 0.7404600977897644, 0.15987607836723328, -0.60781329870224, -0.2705962061882019, 2.021860361099243, -0.1251608282327652, 1.070538878440857, -0.8484834432601929, 0.4056410789489746, -0.03261605277657509, 0.6655885577201843, -1.0622785091400146, -1.4610044956207275, 0.06703191995620728, -0.027447566390037537, 0.9338600039482117, -0.8204906582832336, -1.3291170597076416, 0.10148579627275467, 1.7332876920700073, 0.036012958735227585, -0.38504758477211, 1.8311446905136108, -1.174011468887329, 0.815760612487793, -0.8328537344932556, 1.654024600982666, 1.5368858575820923, 0.832566499710083, -1.2839727401733398, -0.2636682391166687, 0.9769484400749207, 0.6781467199325562, 1.414144515991211, -0.14269696176052094, -0.954245924949646, -0.22858993709087372, -0.6187891364097595, 1.2078273296356201, -2.012552261352539, 0.9550368189811707, -1.0042909383773804, -0.43543529510498047, 0.614503800868988, 0.8571505546569824, -1.4008897542953491, 1.9177428483963013, -1.4722212553024292, 0.8349295854568481, -0.0020725273061543703, -0.13270890712738037, -0.11012088507413864, -1.0465766191482544, -0.9575805068016052, -0.21796199679374695, 0.38028451800346375, -0.7961642146110535, 1.6428143978118896, -0.7305194735527039, -0.8930944800376892, 0.9331536293029785, -1.5816318988800049, 0.3207910656929016, -0.8141510486602783, 0.7817807197570801, -0.023703569546341896, -0.5400110483169556, -0.47559890151023865, -0.09205856174230576, -1.4574626684188843, -0.4370265305042267, 0.4921092092990875, -0.9409462213516235, 0.0676480084657669, -0.8952999711036682, 1.5313539505004883, 0.37136244773864746, 0.48775729537010193, -0.5774794816970825, -1.3711785078048706, 0.23309758305549622, -1.8392279148101807, 0.7415472269058228, -1.5298784971237183, -0.3532326817512512, -1.0728093385696411, 1.369407057762146, 0.7453040480613708, 0.988364040851593, 0.27812501788139343, -0.8482487797737122, 0.28001028299331665, 0.1432064324617386, 0.4039936661720276, -0.9451209902763367, -0.04486037418246269, -0.6225103735923767, 0.36066383123397827, -0.8837452530860901, 2.2421445846557617, 1.1952574253082275, -0.02988787554204464, 2.3679418563842773, -0.3815196752548218, 1.3697699308395386, -0.2633614242076874, 2.9526917934417725, 0.8813351988792419, 0.5517677068710327, 0.4816589057445526, -0.4428693354129791, -0.18148671090602875, 1.497053861618042, -0.046266864985227585, -0.61257004737854, -0.4812937378883362, 1.3249174356460571, 0.9383273124694824, -0.24182525277137756, -1.2320538759231567, 0.00901039782911539, 0.8818622827529907, 1.6813511848449707, 0.9989794492721558, 0.5923606753349304, -0.1323232352733612, -1.9266289472579956, 0.4148634970188141, 0.671952486038208, 0.40175482630729675, 0.5513256192207336, 0.9493369460105896, 1.3577512502670288, -0.1917872577905655, 1.2788848876953125, 1.8344160318374634, -2.1205058097839355, 0.07851222902536392, -0.530276358127594, 1.0518450736999512, 1.5798308849334717, 1.0428541898727417, -0.05100178346037865, -1.3814775943756104, 0.5442612767219543, -0.6926948428153992, -0.27203017473220825, -0.3610125482082367, -1.302903652191162, -0.5854063034057617, -1.2651066780090332, 1.7395437955856323, 0.4348996877670288, 0.38511526584625244, -2.632695198059082, 0.5409639477729797, 0.06401141732931137, -0.04995984584093094, 0.3807010054588318, -0.5720102787017822, -0.9532997012138367, 0.3558782935142517, -0.3177790939807892, 0.19888024032115936, 0.3409346640110016, 1.4840120077133179, -0.5872305631637573, 0.953946590423584, -0.004412026144564152, -0.45043253898620605, 0.6978078484535217, 0.5816656351089478, -0.07109076529741287, -0.3935486376285553, 0.6468886137008667, 0.5477946400642395, 0.8153837323188782, 0.5256317853927612, 0.34104886651039124, 0.1305147409439087, 0.5224930644035339, 1.4448425769805908, 1.7176004648208618, -0.22047755122184753, 0.47002020478248596, -0.41352930665016174, 0.58670574426651, -1.7400821447372437, -2.000379800796509, 0.19301439821720123, -1.0752995014190674, -1.2329819202423096, 1.5600162744522095, 1.3772188425064087, -0.3833700716495514, -0.06138171628117561, -0.6641845107078552, 0.9412396550178528, -0.9819749593734741, 0.058570895344018936, -0.7419354915618896, 0.4531744122505188, 0.696201503276825, 0.33424606919288635, 0.34194499254226685, -0.6515266299247742, 0.4433274567127228, 0.18564732372760773, 0.5380769371986389, -0.7001976370811462, -0.0626886785030365, 1.143702745437622, 0.4566478431224823, -1.7939705848693848, -0.00802661757916212, -0.15702278912067413, -0.5116978883743286, 0.13020864129066467, -2.0820202827453613, 0.04100894182920456, 0.8526857495307922, -0.838201105594635, 0.7415248155593872, 1.290780782699585, 1.340327262878418, 0.1270648092031479, -0.4197993874549866, 1.3045860528945923, -1.705775260925293, 0.4164331257343292, 0.18385548889636993, -0.30424097180366516, -0.9985790848731995, 0.18438410758972168, -0.48272591829299927, 0.12822484970092773, -0.03827979043126106, -0.44902241230010986, -0.9630494713783264, 0.7343745231628418, 0.4307575523853302, 1.263076663017273, -0.37414002418518066, 0.326486736536026, 1.1910839080810547, -0.39713233709335327, 0.044106751680374146, 0.3643365800380707, -1.1153713464736938, -0.6854161024093628, -0.18909898400306702, -1.4094499349594116, -0.5688686370849609, -0.11615359038114548, 0.9333507418632507, 0.6396010518074036, 0.41710567474365234, -0.8400843143463135, -1.322571873664856, -2.806709051132202, 0.17267513275146484, 0.803975522518158, 0.815352201461792, -0.2106798142194748, 0.32551631331443787, -0.7424641847610474, -2.3274013996124268, -0.39018529653549194, 0.41335955262184143, -0.7765375971794128, -0.8729466199874878, -0.06291736662387848, -0.44677045941352844, 1.1439135074615479, 0.8517506122589111, 0.39312705397605896, -0.8150008916854858, -1.2059767246246338, -0.6383183598518372, 1.349952220916748, 1.0461235046386719, -0.7449228763580322, -0.5347731113433838, -0.3814264237880707, 1.2350856065750122, -0.6621562242507935, 0.038693830370903015, 0.48802822828292847, -1.7331585884094238, -1.251062035560608, -1.1122158765792847, 0.941100001335144, 0.7687039375305176, 1.1421146392822266, 0.9610121846199036, -0.057845890522003174, -0.3122171461582184, 2.010019540786743, -0.10746730118989944, 1.1762317419052124, 0.7272300720214844, -0.8763648867607117, -0.8151827454566956, 0.03771377354860306, 2.7076032161712646, 1.146276831626892, 2.0933783054351807, -0.7077325582504272, -0.5897672772407532, -0.14985667169094086, 0.015227283351123333, -0.8631095290184021, 0.7569831609725952, 0.7441086173057556, -0.2262798696756363, -0.712256133556366, 0.8866595029830933, 0.23914849758148193, -1.5841370820999146, -0.357179194688797, 0.03806358575820923, -0.43354812264442444, -0.5648476481437683, -0.8675788640975952, 0.6964117884635925, -0.9186943769454956, 0.28170061111450195, -0.5618027448654175, 0.28777724504470825, -1.0420516729354858, 0.26945891976356506, -0.3630976378917694, 0.6776520013809204, 0.16763515770435333, 0.5581192970275879, -1.0471216440200806, 0.11835265159606934, 0.20043009519577026, -0.8120314478874207, -2.3457529544830322, 0.057723525911569595, 0.13015824556350708, -0.13297134637832642, -0.04324069619178772, 1.448441982269287, -0.9642425775527954, 0.5735388398170471, 0.37473294138908386, -1.601836085319519, -0.7606503367424011, 1.4820740222930908, -0.8193193078041077, -1.3414227962493896, 0.9293043613433838, -0.7054646611213684, 0.5952792167663574, 0.2720620334148407, 0.594616174697876, 0.05959763005375862, 1.2883800268173218, -0.7206231951713562, 0.9629166722297668, -0.24372901022434235, 0.7628931403160095, -0.2028166949748993, -1.200020432472229, 0.009248674847185612, -0.3970572054386139, -0.6060107350349426, -0.18420201539993286, 0.38387128710746765, 0.9866289496421814, 0.708582878112793, 0.9196200966835022, 0.9327735304832458, 0.41079288721084595, -0.4412466585636139, 0.42135778069496155, -1.039285659790039, 0.5564766526222229, -0.9012961983680725, 0.6195715665817261, -1.882709264755249, 0.188408762216568, -1.3378846645355225, -0.4735226333141327, 0.5234178304672241, 0.4035622775554657, -0.09572558104991913, 1.2306265830993652, 1.247967004776001, -1.806983232498169, -0.49451324343681335, 0.9695226550102234, -0.18435119092464447, -0.06241699680685997, 1.0167585611343384, 0.3083273470401764, 2.3983545303344727, -0.010037694126367569, 0.6846660375595093, 1.1735774278640747, 0.16249729692935944, -0.4923171103000641, -0.9665382504463196, 0.4282297194004059, -0.10605672746896744, 1.2460064888000488, 0.14302432537078857, 0.24413880705833435, 1.6966711282730103, -1.657365083694458, -1.1130707263946533, 1.1087332963943481, 1.183754324913025, 1.0051995515823364, -1.1006635427474976, 0.7413245439529419, 0.40749284625053406, -0.12212631106376648, -0.8235224485397339, 1.3916821479797363, 0.13853971660137177, -0.04086943343281746, -0.16769327223300934, 0.6390882730484009, -0.07077118009328842, 0.15143710374832153, 1.3156243562698364, -0.36621975898742676, -0.5826055407524109, 1.221962809562683, 1.4023048877716064, -1.199852705001831, 1.2525407075881958, -0.17799516022205353, 2.176177978515625, -0.7760747671127319, -0.6170791387557983, 0.3771825432777405, -0.31704339385032654, 1.5342273712158203, -0.28241288661956787, -0.11978869140148163, 0.8735467195510864, -1.2278881072998047, 0.5746042728424072, -1.6568777561187744, 0.27624937891960144, 1.1955968141555786, 0.8272745609283447, 0.850917637348175, 1.7386330366134644, -0.5243723392486572, 1.958943247795105, 0.0037511298432946205, -1.175503134727478, 1.0367732048034668, -0.49207964539527893, -0.39454537630081177, 0.7561478614807129, 0.2853092551231384, -0.5827335715293884, -0.3593246042728424, -0.24444791674613953, 0.863259494304657, 0.4240312874317169, 1.9688247442245483, 1.2618777751922607, 1.9868953227996826, -0.3309359848499298, -1.7642368078231812, 0.26378971338272095, 1.256899356842041, -1.3960742950439453, 1.1558598279953003, -0.8095535039901733, 0.11133073270320892, 1.6702648401260376, 1.2040210962295532, -1.5497699975967407, -0.3243376612663269, 0.7797338962554932, 0.6752640604972839, 2.1596460342407227, -0.020849736407399178, 0.40095534920692444, -0.17472627758979797, -1.3212223052978516, 0.6154279708862305, -0.005394160281866789, -0.0647570937871933, -0.5336366891860962, -0.8513162136077881, 2.2687509059906006, 1.5765647888183594, -2.347930431365967, 0.3055943548679352, 0.35946112871170044, 0.26273447275161743, 0.7360596060752869, 1.2471636533737183, -0.3038617968559265, -1.437694787979126, -1.3171279430389404, 1.512297511100769, 0.575497567653656, 0.6082251071929932, -0.17996099591255188, 1.6528092622756958, 0.2551531195640564, -0.6974638104438782, 0.7534221410751343, -0.08998309820890427, 0.8366634845733643, -0.02600858546793461, -0.07402783632278442, -1.3419758081436157, 0.5732945203781128, -0.8477500677108765, 0.2734104096889496, 0.220319926738739, 0.4774256944656372, -0.22094030678272247, -0.9726256728172302, 1.2331759929656982, 0.9037920236587524, 0.021023716777563095, -0.4207145869731903, -0.650705873966217, 0.6941226720809937, -0.28203362226486206, -1.3625901937484741, -1.8933351039886475, -0.5751640796661377, -1.113928198814392, 0.9867660999298096, 0.5861059427261353, 0.5627792477607727, 1.3818811178207397, 0.6468928456306458, -0.7159239649772644, 1.294478178024292, 0.5249150991439819, -0.1804475039243698, -0.1421670764684677, 0.7537264227867126, 1.077289342880249, 0.06918301433324814, 1.4190492630004883, 0.6068430542945862, -0.48938220739364624, -0.02724572829902172, -0.8465855717658997, 0.1372208446264267, 1.2076456546783447, -1.282179594039917, 0.6285771727561951, 1.35248601436615, 0.4645381271839142, -1.0358952283859253, 0.6504817605018616, 0.15737879276275635, -1.3767614364624023, 0.5382622480392456, 0.9289586544036865, 2.0394816398620605, -0.6370421051979065, 2.0856149196624756, -0.5568260550498962, -0.5044042468070984, -1.939571738243103, -0.30657482147216797, -0.5071814656257629, 1.0474090576171875, 1.263993263244629, 1.1409127712249756, 0.23780223727226257, -1.5317797660827637, 1.050894856452942, 0.2736961245536804, 0.01676096022129059, 1.7723760604858398, -0.02736741304397583, -0.14384767413139343, 0.516818106174469, 0.5569167137145996, 0.189638152718544, -0.2154087871313095, 0.2636285126209259, 0.5919983386993408, -0.7286537885665894, 1.4357097148895264, 0.8712601065635681, 0.28295326232910156, -0.6575936079025269, -1.2196711301803589, 0.49435943365097046, -0.04276967793703079, 2.0840604305267334, 0.3970201313495636, 1.045681357383728, 1.4277780055999756, 0.45796164870262146, 2.050654649734497, -0.4303101897239685, -1.032843828201294, -1.3081605434417725, -0.15781304240226746, -0.5905329585075378, -1.3198293447494507, -0.7977198958396912, 2.1938984394073486, 0.796523928642273, 0.1917230784893036, 0.47444137930870056, -0.05863919481635094, -0.7567557692527771, 0.42438122630119324, 0.8273571729660034, -0.8507793545722961, -0.1223885789513588, 1.2338944673538208, 0.8018617033958435, -0.9375429153442383, 1.2690144777297974, 0.33042460680007935, 1.0013056993484497, 1.0164514780044556, -0.4542704224586487, 0.7195011377334595, 1.3235043287277222, -1.8212120532989502, 0.8561845421791077, -1.9664353132247925, -0.002112023066729307, -0.10383386164903641, 0.40258827805519104, 1.0191022157669067, -1.3567352294921875, 0.32873421907424927, 0.8793315887451172, -1.7170023918151855, -0.4717392921447754, 0.24972420930862427, 0.018947167322039604, 0.5087318420410156, -0.7783183455467224, 0.6762726902961731, -0.9814937710762024, -2.295513868331909, -0.38349485397338867, 1.0530266761779785, -1.3032746315002441, -1.4733502864837646, 1.6337335109710693, -2.2192347049713135, 0.7218294739723206, -0.8995405435562134, 0.24717754125595093, 0.7477686405181885, -1.8095561265945435, -0.5583372116088867, 0.7700580358505249, -0.8961204886436462, 0.6709347367286682, 0.10452277958393097, 1.7109630107879639, -0.38752442598342896, 1.1903198957443237, -1.0827854871749878, -0.8811695575714111, -1.160591959953308, 1.4863383769989014, -0.2985304296016693, -0.8985070586204529, 1.5078905820846558, 0.266409695148468, 1.5603983402252197, -0.5230238437652588, -0.1866784691810608, 1.4213272333145142, 0.18790635466575623, 0.38610199093818665, -0.5607926249504089, -0.13243363797664642, 0.22516268491744995, 0.6842875480651855, 0.007657863199710846, -1.106572151184082, -0.02394767850637436, -1.002095103263855, -0.23685823380947113, -2.8579213619232178, 1.3533447980880737, -1.0006660223007202, -0.4828302264213562, -0.5050649642944336, 0.6302633881568909, -0.26107296347618103, -0.5815623998641968, -1.7615306377410889, 0.8402713537216187, -1.6553007364273071, 0.0817314088344574, -0.4738026559352875, 0.1869470477104187, -1.1141144037246704, 1.744234561920166, 0.15980830788612366, -0.39387792348861694, -0.1244146004319191, -0.787395715713501, -1.7435866594314575, 0.2725795805454254, -1.0332605838775635, 0.7603990435600281, -0.2691841721534729, 0.7974588871002197, 0.4463396668434143, -1.0185129642486572, -0.7893991470336914, 0.4557999074459076, -0.32809582352638245, -0.40777087211608887, -0.882818877696991, 1.921808123588562, 0.058832865208387375, 0.4954540431499481, 0.21217714250087738, 1.113952398300171, -0.44030556082725525, -0.37208887934684753, -0.3179463744163513, -1.124802589416504, 1.4889094829559326, 1.4381721019744873, -0.9147716760635376, -0.7900915145874023, -0.7952115535736084, -0.6200568079948425, 0.22913579642772675, -0.815115213394165, -1.7512129545211792, 1.1969337463378906, 0.5009135603904724, -1.6591358184814453, 1.0239686965942383, 1.0609034299850464, 0.20244470238685608, 0.23498769104480743, -0.7555624842643738, 0.47127100825309753, 1.1294981241226196, 0.14339828491210938, 0.9062420725822449, -1.3997641801834106, -0.8726619482040405, -0.21238262951374054, -0.7872068285942078, -0.2450627088546753, -0.834014892578125, -0.9020562171936035, 0.9226900935173035, 0.10274305939674377, -0.9590861201286316, 0.42201098799705505, -0.6476867198944092, -0.9950173497200012, -1.1009809970855713, 1.227168083190918, -0.7808517217636108, -1.1821953058242798, -1.4063929319381714, 0.6594264507293701, 0.359836220741272, 0.6508097648620605, -0.0648287907242775, 0.4311760663986206, 1.6843538284301758, 0.1351100653409958, -0.0761333703994751, 0.9573599100112915, -1.0780575275421143, 0.46431201696395874, 0.20136983692646027, 0.17718255519866943, 0.5908383131027222, 0.7463293671607971, 0.3846406638622284, 0.10525355488061905, 1.0416933298110962, -0.7208669781684875, -0.2875104248523712, -0.06686951965093613, -0.5837838649749756, 0.20863574743270874, 0.1592748910188675, -1.5212836265563965, 1.056588888168335, 0.6253430247306824, 0.8456527590751648, -0.35468238592147827, -1.1928272247314453, 0.30256178975105286, -1.491262435913086, -1.2695168256759644, -0.14433254301548004, -2.5681369304656982, 0.654847264289856, -0.7388754487037659, -0.6056039929389954, -0.10785873979330063, 0.4686146378517151, -1.1971076726913452, 1.3212865591049194, -1.046413779258728, 0.7656695246696472, -0.2965161204338074, 0.6538439393043518, -0.7910274863243103, 0.24591456353664398, -0.8613685965538025, -1.1501559019088745, -0.8778127431869507, 0.3352131247520447, 0.7747799158096313, 0.0010020160116255283, 0.6952851414680481, -1.286453127861023, 0.9664139151573181, -0.6790744066238403, -0.7466079592704773, 0.714483380317688, -0.09443633258342743, -0.3205567002296448, -1.2347453832626343, 1.1240330934524536, 0.5727078318595886, 0.24397747218608856, -0.18853521347045898, 0.009536885656416416, 0.14794921875, 0.374436616897583, 0.14589332044124603, 0.37833791971206665, 0.55018550157547, 2.5089762210845947, 1.2663898468017578, 0.3660641312599182, -1.0682475566864014, 0.30898791551589966, -0.6444072723388672, -0.6952475905418396, 0.7014667391777039, -1.3841538429260254, 0.54688560962677, -0.9255220890045166, 1.3906956911087036, 1.4924482107162476, 0.9468843340873718, -1.0856751203536987, -0.27755799889564514, -0.7603863477706909, 1.224657654762268, 0.1498815268278122, -2.661808729171753, -1.4910892248153687, -1.010598063468933, -0.26606622338294983, -0.5626088976860046, 0.11317852884531021, -0.09829756617546082, 0.6017816066741943, 0.5777137875556946, 1.6772211790084839, 1.9326494932174683, 2.1247754096984863, 1.8631376028060913, -0.9101490378379822, -1.6064367294311523, 0.5299452543258667, 0.18969981372356415, -0.7216814756393433, -2.1004817485809326, 0.5646388530731201, -0.02949080988764763, 0.9082649350166321, 0.31254827976226807, 1.8314547538757324, -0.4309203624725342, 0.1163298636674881, 0.09753727167844772, 0.9066964983940125, 0.19102546572685242, 1.6509547233581543, 0.8222209811210632, 0.38379836082458496, 1.2073569297790527, -1.2487033605575562, 0.5007452368736267, 0.20564401149749756, -0.7909513115882874, -0.2998007833957672, 0.6795732975006104, -0.5650932192802429, -0.01920905150473118, -0.35159772634506226, 0.49653568863868713, -1.629093050956726, 0.7941891551017761, -0.5333626866340637, -1.3965781927108765, 0.567098081111908, -1.4226821660995483, 1.1524055004119873, 0.5302225351333618, 0.7753446102142334, -0.707068920135498, 1.595586895942688, -0.48845911026000977, 0.12960799038410187, 0.552821934223175, -0.42295700311660767, 2.089064598083496, 0.057282064110040665, -0.5566138029098511, -0.8303237557411194, -1.6372463703155518, 0.31583189964294434, -0.2611987888813019, -0.6229043006896973, 0.6767932176589966, 0.9754725098609924, 0.28371742367744446, -1.4898992776870728, 0.29123497009277344, -0.4056055247783661, 0.06117631494998932, 1.4787907600402832, 2.1572153568267822, -0.5349280834197998, 0.07775387912988663, -0.35426485538482666, 0.304127037525177, -1.8051272630691528, 0.3802456557750702, -0.9889448881149292, -1.1659059524536133, 0.6580734252929688, 0.7840456366539001, -0.5069587826728821, 0.7717868685722351, -1.9172979593276978, -0.37736114859580994, 1.5203288793563843, 0.722532331943512, 0.7976427674293518, -1.1319524049758911, -1.0771101713180542, -0.6880845427513123, -0.07935401052236557, -0.6167700886726379, 0.5401305556297302, 1.1730269193649292, -0.5779480934143066, 1.5804574489593506, 0.44583818316459656, -0.7510301470756531, -1.4066578149795532, 0.3188175559043884, 0.30230310559272766, -0.7682546377182007, 0.47441500425338745, 0.36808159947395325, -0.6524838805198669, -0.18648867309093475, 0.15841132402420044, -0.17173753678798676, -1.3876250982284546, -1.0085296630859375, 1.0981374979019165, -0.6072736978530884, 0.7464823126792908, 0.8595427870750427, -0.4632927179336548, 2.0129778385162354, 1.1016976833343506, -1.2285754680633545, 1.6888401508331299, -0.24529510736465454, 0.024341218173503876, -0.16098862886428833, -1.471697211265564, 0.10556800663471222, 0.4365686774253845, -0.08747469633817673, -1.195595145225525, 1.4946091175079346, -2.4966001510620117, -0.41322508454322815, -0.03370010107755661, 1.9227125644683838, 0.14756804704666138, 0.8100153803825378, 0.4002739489078522, -0.7369877099990845, 0.4887143075466156, 0.8916969895362854, 0.6788318157196045, -1.1135309934616089, 1.4133280515670776, 0.1021890714764595, -0.48741158843040466, -1.3128166198730469, 0.44334304332733154, 1.3180445432662964, 0.24401983618736267, 0.31043094396591187, 0.1647697538137436, -0.1725464165210724, -1.2745002508163452, 0.7033622860908508, 0.7126272916793823, -0.7139950394630432, 0.7708525061607361, 0.72552490234375, -1.5214635133743286, -0.28585317730903625, 0.9923873543739319, -1.353481411933899, 1.7019603252410889, 0.6213769316673279, -0.8646239638328552, 0.27978238463401794, 0.2265085130929947, -0.5760381817817688, 0.6385385394096375, 0.052016451954841614, -0.3451295793056488, -1.1102197170257568, -0.5340981483459473, -0.17505869269371033, 1.028337001800537, 0.869013786315918, -0.5393466949462891, 0.04205729067325592, -1.2185988426208496, -0.2415771186351776, 2.6114401817321777, 0.7513322830200195, -0.9056969285011292, -1.6986557245254517, -0.7414001822471619, -0.1395488828420639, -1.4162139892578125, 0.17042435705661774, -1.135567307472229, 0.6035904884338379, -0.1557467132806778, -0.14709414541721344, -0.7002960443496704, 0.8165602087974548, 1.1210473775863647, 1.6294740438461304, 0.46519193053245544, 0.4999135434627533, -0.5121997594833374, -0.5413545370101929, -0.34679412841796875, -0.6685091257095337, -2.2676570415496826, -0.345121830701828, -0.15180683135986328, -0.39854228496551514, 0.8988651633262634, 1.2999539375305176, -0.6579527258872986, 1.1433045864105225, 1.2491086721420288, -0.9113154411315918, -0.1269284337759018, -0.3867562413215637, 0.8627190589904785, -0.6283418536186218, 0.2934935688972473, 0.6940237283706665, 0.9247086644172668, 0.9884183406829834, 1.1505523920059204, -0.12119433283805847, -0.7766507863998413, 1.387234091758728, 0.32161590456962585, -0.8219866156578064, -0.0052973986603319645, 2.007871389389038, -1.3657329082489014, -0.6672674417495728, -1.0481687784194946, 1.2218880653381348, 0.7573264241218567, 0.5358871817588806, -0.05039310082793236, 0.5089477300643921, -0.3842253088951111, 0.11627514660358429, -0.3852185010910034, 0.03424558788537979, -2.1588053703308105, 0.48955103754997253, 0.47482332587242126, 1.4983160495758057, 0.28308939933776855, 0.4955582618713379, 1.1294904947280884, 0.6430730223655701, 0.16388651728630066, -0.6091916561126709, 0.8204978704452515, 2.503141164779663, -1.0344558954238892, 1.5130475759506226, 1.61299467086792, -0.012822034768760204, -0.3899449110031128, 0.12488735467195511, -1.4721652269363403, 0.30996760725975037, -2.102670192718506, -0.8120834231376648, 0.4499457776546478, 0.0057775420136749744, -1.3666855096817017, -0.24205070734024048, 0.813768208026886, -0.31154996156692505, -1.7738163471221924, 1.4154938459396362, -0.22874192893505096, -0.23516744375228882, -0.19115257263183594, 1.0079569816589355, 0.8650625348091125, -0.9624125361442566, -0.8980762362480164, -0.11804517358541489, -0.7886472344398499, 0.770574152469635, 0.9630460739135742, 0.1448492705821991, 2.219447374343872, -1.6796596050262451, -1.894059419631958, 0.121049664914608, 0.19871874153614044, 0.8348866105079651, -1.8035207986831665, 0.3886614739894867, 1.147547960281372, -1.2219573259353638, -0.6075936555862427, 1.4947288036346436, -1.259173035621643, -1.00810968875885, -0.25760406255722046, 0.6630613207817078, -0.8695697784423828, 0.27424484491348267, -0.8296306133270264, -0.05799521878361702, 0.515083909034729, -0.24957266449928284, 1.2934397459030151, 1.6295297145843506, -0.5044779777526855, -0.03193432837724686, 1.3068073987960815, -0.4182208180427551, -0.14034724235534668, 0.6186515092849731, -0.3568520247936249, 1.1495482921600342, 1.1151418685913086, -0.9417270421981812, -0.38251522183418274, -0.3106653094291687, -0.5824573040008545, -1.1655423641204834, -0.684026837348938, 2.427442789077759, -0.2734311521053314, -0.740669310092926, 0.4584190249443054, -2.6121723651885986, -0.4905577301979065, 0.9375671148300171, -0.639946699142456, 0.31512463092803955, -0.9733035564422607, -1.7129930257797241, 1.038091778755188, -0.7158247232437134, 1.108515977859497, 1.1747641563415527, 0.5765145421028137, -1.63741934299469, 1.1827863454818726, 2.1040897369384766, -0.398519366979599, 0.023808948695659637, -0.8012300729751587, 0.884407103061676, -0.4411909878253937, 0.5670541524887085, -0.6170486211776733, -1.1516733169555664, 0.025016319006681442, -0.6842013597488403, 0.05273551121354103, 0.7629727125167847, -0.1390582025051117, -0.7512024641036987, 1.1328483819961548, 0.7487834095954895, -0.6528141498565674, -0.29670462012290955, 1.122300148010254, 0.18682150542736053, 0.5906379818916321, -0.3386070728302002, 2.627149820327759, -0.1716948002576828, 0.5052282214164734, 0.7024074792861938, -0.7211119532585144, 1.5658190250396729, 0.6909517645835876, 0.07840912789106369, 0.5208868980407715, 1.081073522567749, -0.06780128926038742, -1.7772793769836426, 0.5139992237091064, 0.05342157930135727, 0.8463438749313354, 0.8282654881477356, 0.4966234266757965, -0.9272811412811279, -0.2075200080871582, -1.6991608142852783, -1.9771910905838013, -0.850207507610321, 1.0203121900558472, 0.29641664028167725, -0.23618926107883453, 1.0135715007781982, 1.23958420753479, 1.3498497009277344, -0.9219691753387451, 0.2757839560508728, 0.5328432321548462, 2.0138027667999268, 0.4207116961479187, 1.162919521331787, 0.08760664612054825, -0.6129361987113953, 0.9444833993911743, 0.8583475947380066, 1.5547573566436768, 1.205552101135254, -0.9595672488212585, 1.3464807271957397, -0.6192066073417664, 0.17235630750656128, -0.4124571979045868, -0.6445180177688599, 0.25414642691612244, 1.5241022109985352, 0.2441813051700592, -1.4141857624053955, 0.3752168118953705, 1.125626564025879, -1.385464072227478, -0.6852383613586426, -0.6771084070205688, 1.1924396753311157, 0.09668850898742676, 1.3957784175872803, -0.07481256127357483, -0.5848400592803955, 0.6374312043190002, 0.3569561839103699, -0.8062191009521484, -0.5516774654388428, -0.03414415940642357, -0.39534875750541687, -0.5215046405792236, 2.466306686401367, 1.8513120412826538, 2.0916266441345215, -0.4230639934539795, 2.1079678535461426, 0.8793771266937256, -0.19039234519004822, -1.622323989868164, -1.4165582656860352, -0.05802397057414055, -1.2440848350524902, 0.11467728763818741, -0.005346913356333971, -0.08224157989025116, 0.6358177065849304, -1.3004511594772339, 1.136110544204712, -0.4977158308029175, -0.4277031123638153, 0.460840106010437, 0.06533748656511307, -0.07018427550792694, 0.9513471722602844, -0.5529805421829224, -2.395596742630005, -0.12536446750164032, -0.19682006537914276, 0.9624678492546082, 0.010668512433767319, -0.06142403185367584, -0.14415842294692993, 0.37874317169189453, -0.9009551405906677, 0.034087155014276505, 1.8016514778137207, -0.266044944524765, 0.8380311727523804, 0.9655921459197998, -0.9145910143852234, -1.9138206243515015, 0.4431403875350952, 2.22430419921875, 0.008778003975749016, 0.4032595753669739, -0.7604554295539856, -0.49245333671569824, -0.2171047478914261, -0.43944305181503296, -0.20042340457439423, -0.5045763850212097, -0.28540071845054626, 0.690582811832428, 0.142875537276268, 0.7273674607276917, -0.25420477986335754, 1.2900540828704834, -1.0677684545516968, 1.543978214263916, -0.7241043448448181, -1.2559798955917358, 0.5531309247016907, -0.8809900879859924, -1.9138845205307007, -1.8779011964797974, 0.3032284379005432, -0.26127275824546814, 0.1952284276485443, -0.7444229125976562, 0.2961858808994293, 1.4184201955795288, -1.0578166246414185, 0.6816287636756897, 0.38646525144577026, 1.0149141550064087, -0.27074068784713745, 2.280158519744873, 0.63633131980896, 0.2610882818698883, -0.21801690757274628, -0.5575251579284668, -1.5111467838287354, 0.3076949715614319, -1.604264736175537, -1.3293448686599731, -0.4780293405056, -0.4365849196910858, -0.11412075161933899, -1.5032702684402466, -1.0374163389205933, 0.6655272245407104, -0.6869649291038513, 1.064350962638855, 1.2012269496917725, 0.6232191324234009, 0.7585224509239197, -0.09416953474283218, 1.2320183515548706, -0.08315480500459671, -0.559657633304596, -0.7144620418548584, 0.4468356668949127, 0.6505120992660522, -0.6158883571624756, -1.8860950469970703, 0.9785823225975037, 0.8503643274307251, 0.8088652491569519, 0.7646925449371338, -0.6514284610748291, 0.11716365814208984, 0.8640218377113342, -0.7149126529693604, -0.24164101481437683, 1.4399607181549072, -1.2310739755630493, -1.1835966110229492, -0.45346856117248535, 1.1961838006973267, -0.14389248192310333, -0.05030215531587601, 0.5226831436157227, -0.3299899101257324, 0.4476339519023895, -1.113020658493042, -0.7075902223587036, -0.8711868524551392, 0.285720556974411, 2.6691300868988037, 0.6255404949188232, -0.5960742831230164, -2.0348165035247803, -0.2629334628582001, 0.39147844910621643, -0.5648026466369629, 0.01092151366174221, 0.5612869262695312, 0.4529109001159668, -0.3637489974498749, 0.36640405654907227, 0.2861405909061432, -1.2952884435653687, 0.9481825828552246, 0.19974792003631592, 1.3498682975769043, -0.5160651803016663, -0.0832185298204422, -0.9002315402030945, 1.6819857358932495, 0.5774943828582764, -1.1677806377410889, 1.0684329271316528, 0.2590074837207794, -1.850629448890686, 1.191599726676941, 0.9022508859634399, -0.06705282628536224, 1.1807546615600586, -0.5716021060943604, -1.6569774150848389, 0.7724812030792236, -0.5353378057479858, -1.0555812120437622, 0.9855092763900757, -0.10666362941265106, -0.5123329758644104, 1.0771552324295044, 1.9979397058486938, -0.3394501805305481, 0.22913525998592377, 0.27947619557380676, -1.2042615413665771, 0.567594051361084, -0.787666380405426, 1.7607879638671875, -0.995938777923584, 0.25795087218284607, 0.33883288502693176, 0.7437608242034912, 0.7969427108764648, 1.6235826015472412, 0.09657857567071915, -0.6947653293609619, 1.2489043474197388, -0.3221375048160553, 0.1563318520784378, -0.04853082075715065, -0.5085383057594299, 0.3685430586338043, 0.23901590704917908, -2.3085076808929443, 0.18367953598499298, -0.841438889503479, 0.2524309456348419, 0.2240126132965088, 0.11227406561374664, -0.7879511117935181, 0.08488903939723969, -0.20798861980438232, -0.07839512825012207, 0.7673499584197998, 0.2145431488752365, -0.7023177146911621, 0.6085174679756165, 0.9345865845680237, -0.5029765963554382, -2.4463579654693604, -0.41774171590805054, 0.7786872386932373, -0.5986257195472717, -0.34051376581192017, 1.0521332025527954, 0.2609499990940094, 1.9174065589904785, 0.8891857266426086, -0.5480706691741943, -0.2980445623397827, 0.6227272152900696, -0.38722601532936096, 0.5213591456413269, 0.055103983730077744, -0.05485105514526367, 1.3739895820617676, 0.8113203048706055, -1.5398201942443848, 1.6013813018798828, 1.7374298572540283, 0.4206610321998596, -0.8983980417251587, 0.9812601804733276, -0.23233604431152344, 0.10826665908098221, 0.25448545813560486, 0.21426762640476227, -1.410695195198059, -0.7798459529876709, -0.21569395065307617, 1.0833443403244019, -0.05135347321629524, 0.32800987362861633, 0.2905471920967102, 0.6121284365653992, 0.5454583764076233, -0.054345015436410904, -1.6181119680404663, -1.6471613645553589, -1.639347791671753, 0.4299962818622589, -0.4004627466201782, -0.4967242181301117, 0.7496278882026672, 2.4697799682617188, 1.0632781982421875, 0.3487832844257355, 0.7924407720565796, 1.6350501775741577, 0.6558563113212585, -0.11157727241516113, -1.0500926971435547, -0.478441447019577, -0.8094555735588074, -2.2384142875671387, -0.7482247948646545, -0.9525847434997559, -0.2033218890428543, 0.024872133508324623, 2.239856004714966, -0.4021437168121338, 1.7772566080093384, 0.5838660597801208, 0.8164334893226624, 1.849027395248413, 1.3561946153640747, -0.17402249574661255, -0.4464717507362366, -0.7309726476669312, 0.6488574147224426, -1.2865031957626343, -1.5186042785644531, -0.16916970908641815, -0.16556021571159363, 0.5693149566650391, 0.9370669722557068, 1.012992262840271, 2.8599326610565186, 0.08171918243169785, 0.5859660506248474, -0.8419895172119141, 1.8090670108795166, 0.18531189858913422, 1.649987816810608, 0.8510854840278625, -1.0842633247375488, 0.035719048231840134, 0.9070826172828674, 1.9184486865997314, 0.06096420809626579, -1.2839770317077637, 0.11859966814517975, 1.062938928604126, -0.16396139562129974, 1.4289093017578125, 1.4817326068878174, 0.8887848258018494, -1.3738385438919067, 0.8764440417289734, -0.252305269241333, -2.903371810913086, 0.00018078032007906586, 0.7970970273017883, -0.04212401062250137, 1.1220042705535889, -2.0096116065979004, -0.8985037207603455, -0.9550327062606812, -1.1559407711029053, 2.5305564403533936, -1.065230131149292, 1.488195538520813, -1.1804958581924438, 0.7570679187774658, -0.08461862057447433, 2.027092695236206, 0.5860982537269592, 2.57258677482605, -0.28482043743133545, 0.02378252148628235, 2.499666452407837, 1.1153205633163452, 2.211881399154663, -0.4063473343849182, -0.3882887363433838, 2.016812324523926, 0.7089775204658508, 0.33988112211227417, -0.4770359694957733, -0.15507888793945312, -0.4388139545917511, -0.9927185773849487, 0.22605283558368683, 0.8216320276260376, 1.0531296730041504, 0.5521709322929382, 0.06881801784038544, -0.1314779669046402, 0.7560476064682007, -0.13924965262413025, 1.3631693124771118, -0.03797043114900589, -0.7435562014579773, 0.16454097628593445, 1.0363900661468506, 0.3663274049758911, -1.2522661685943604, 0.4601316750049591, -0.3251294195652008, 0.09824297577142715, -1.221531629562378, -2.0517520904541016, 0.24554021656513214, 0.037483420222997665, -1.4494659900665283, -0.06018538028001785, 0.7829110026359558, 0.4752705991268158, 0.0831747055053711, -0.2689829170703888, -0.06245610862970352, 1.2753419876098633, -2.223367214202881, -1.0179632902145386, -0.1263866126537323, -0.27624472975730896, -0.28481170535087585, -0.7574217915534973, 1.645514726638794, 0.2925150990486145, -0.669626772403717, 1.2324672937393188, -1.6879369020462036, 0.013773390091955662, 0.6719191074371338, 1.1596297025680542, 0.22237224876880646, 0.008904341608285904, -0.8987409472465515, 0.8048049211502075, 1.4068176746368408, -0.005460570566356182, 1.5776424407958984, 0.4384407103061676, 0.47600704431533813, -0.716852605342865, -0.5113903284072876, 0.49078452587127686, -1.37493097782135, 0.11999012529850006, -0.8922358155250549, 0.5438456535339355, 0.22556646168231964, 0.3465864062309265, 0.4247962534427643, 2.0478971004486084, -1.2879139184951782, 0.6694046854972839, 0.20921486616134644, 0.3800303637981415, 0.14938382804393768, 0.4534095525741577, 0.6180819272994995, 0.32049375772476196, 0.536372184753418, 0.026180163025856018, 0.10628185421228409, -2.7976632118225098, 0.43568843603134155, 1.013474941253662, 1.3704309463500977, 2.529782772064209, -0.31030604243278503, 1.712027907371521, -0.22281625866889954, 1.249528408050537, -1.4442423582077026, -0.4319603443145752, -0.8950462341308594, 0.38888171315193176, 0.0916171744465828, 0.6758272647857666, -2.0049407482147217, 1.505411148071289, -0.8224878311157227, 0.7234723567962646, 0.32507508993148804, -1.6205805540084839, -1.1223998069763184, 0.37125909328460693, -0.9721566438674927, 0.23405708372592926, 1.0905545949935913, 0.994468629360199, 0.6228450536727905, 0.6033671498298645, -2.4688074588775635, 0.8739718198776245, 0.4123896360397339, 1.5818352699279785, -1.4614921808242798, 0.2515465319156647, 0.16672983765602112, 0.1465902030467987, 1.5736767053604126, 1.0351449251174927, 0.6843129396438599, -0.6081520915031433, 0.3069026470184326, -1.0061349868774414, -1.4447824954986572, 0.7033443450927734, 1.5626755952835083, -0.3022352457046509, -0.1650122106075287, 0.6322342753410339, 1.6739333868026733, -0.02028302103281021, -0.5162420868873596, -1.4279004335403442, -0.299238383769989, 1.3108325004577637, -0.29454222321510315, 0.95998615026474, -1.3810486793518066, -1.6343698501586914, -0.22879669070243835, 0.4306464195251465, 1.4804760217666626, 0.007738584652543068, -0.9572122693061829, -0.40180256962776184, 0.3807116746902466, -1.4224231243133545, 0.48220938444137573, -0.1574966460466385, 0.7952176332473755, 0.794701874256134, 0.7991992831230164, -0.7625656723976135, -0.6206793189048767, 0.10253911465406418, 0.32397228479385376, -0.9196767807006836, -0.37980538606643677, -0.1974094659090042, 0.07757992297410965, -1.013810396194458, -0.7262937426567078, 1.241051197052002, -1.0436044931411743, 0.9639902114868164, -0.1564224660396576, 0.9064342379570007, -0.031160853803157806, 0.0633024349808693, 0.8594557642936707, 1.61154305934906, -0.3622802495956421, 2.6112112998962402, 1.6623326539993286, 0.06236949935555458, -2.1467764377593994, 0.5219113826751709, 1.189200758934021, -1.4700591564178467, 0.5925906300544739, 0.7328028082847595, 0.5683050751686096, 0.44362393021583557, -1.1539890766143799, 0.5167896151542664, 0.27839285135269165, -1.7858736515045166, -0.3119576573371887, -0.5506428480148315, -0.25746509432792664, -1.5834252834320068, -0.20487990975379944, -1.5528686046600342, -2.600661039352417, 0.012447734363377094, 1.2415440082550049, 0.03192242234945297, 1.4560859203338623, -1.0200634002685547, 1.099642038345337, 0.4473036229610443, 0.047900956124067307, -0.2687511444091797, 0.8351670503616333, 1.875666618347168, -1.1614900827407837, 0.7701984643936157, 0.8060157895088196, -0.45864495635032654, 1.0098589658737183, 0.4970560371875763, 0.7648541927337646, -1.759533405303955, -0.5864958167076111, 0.6183972954750061, -0.39120006561279297, 1.1877840757369995, 1.5548090934753418, 0.07871323078870773, -0.13117927312850952, -0.29282423853874207, 0.7769613265991211, -1.041187047958374, -0.007270451635122299, -1.5505982637405396, 1.00113844871521, -0.9694576263427734, 1.081750750541687, -1.0655484199523926, -0.855771541595459, -0.65781569480896, -0.9366276264190674, -0.049470312893390656, -0.7579341530799866, -1.399122714996338, -0.6918571591377258, -0.3167233169078827, 0.8534337282180786, 0.692475438117981, 0.3545611798763275, 0.2100258469581604, -0.46948713064193726, -0.2241654247045517, 0.8861612677574158, 0.13399749994277954, 0.4860724210739136, -0.6504651308059692, -1.621557593345642, 0.668989896774292, -0.3623150587081909, -0.31394633650779724, -0.34858784079551697, 0.09550491720438004, 0.9046915769577026, 0.01576821692287922, -0.24816083908081055, -0.2973535358905792, -2.0747084617614746, 0.696672797203064, 2.1645641326904297, -0.6678482294082642, 2.665205955505371, 1.6443793773651123, 0.37021148204803467, 0.3374978005886078, -0.4929603040218353, 0.2999075949192047, 0.8806672096252441, 0.07771140336990356, -0.548431396484375, -0.29592621326446533, -0.9232370257377625, 0.7841833829879761, 0.9980716109275818, -0.8259821534156799, -0.315967321395874, 0.09549453109502792, -1.420669674873352, 0.9971585869789124, -0.04874397814273834, 0.5662195086479187, -1.4172755479812622, -0.23757760226726532, 1.194424033164978, 0.3829309642314911, 0.21290212869644165, -0.906752347946167, 0.20459988713264465, -0.40809839963912964, 2.2479774951934814, -0.6348935961723328, -0.0992296114563942, -1.8624013662338257, -0.45368146896362305, 0.4800267517566681, 0.30895546078681946, 0.5600221753120422, -0.36309313774108887, 1.1189689636230469, 0.8347931504249573, -0.4113631248474121, -0.12315822392702103, -0.43007561564445496, -0.38083750009536743, 0.22623617947101593, 0.5927902460098267, 0.49597692489624023, 0.8460066914558411, 2.0147132873535156, -1.0327708721160889, 0.5276902914047241, 0.3048853874206543, -2.1076548099517822, -1.596392273902893, -0.4799268841743469, 1.104298710823059, -0.356724351644516, -0.6335887312889099, 0.036262013018131256, 0.8385008573532104, -1.2011346817016602, -0.6734592914581299, 0.8993422389030457, 0.7727454304695129, 0.5732774138450623, -1.0454537868499756, -0.7804323434829712, -0.09564325958490372, -1.1126954555511475, -0.5707900524139404, 0.23652391135692596, -0.5254127383232117, 2.5853993892669678, -0.3916066288948059, -1.335341215133667, 0.5946579575538635, -1.5102602243423462, 1.64426589012146, -0.17066100239753723, 0.6695809960365295, 0.5285061001777649, -0.282939076423645, 0.6558639407157898, -1.625938057899475, -1.7308266162872314, 0.268060564994812, 0.207771897315979, 2.4135422706604004, -1.8988803625106812, -0.09899517893791199, -0.046976491808891296, -0.3581232726573944, -3.2254786491394043, -0.6854640245437622, 0.1452929526567459, -0.20005646347999573, 2.155644416809082, 0.9472377300262451, -1.3743841648101807, 0.23524680733680725, -1.4352421760559082, -0.5276714563369751, 0.2840936481952667, -0.702545166015625, -0.14763890206813812, -1.123367190361023, -0.5640087127685547, -0.5158500671386719, -1.061307430267334, -0.47129952907562256, 1.0191209316253662, 1.4683656692504883, -0.9346098303794861, 1.2900618314743042, -0.07098262012004852, 0.020931385457515717, -1.1857354640960693, 1.5365393161773682, -0.4838206171989441, -0.07694276422262192, 1.8988256454467773, 1.3294000625610352, -1.2649420499801636, -0.6612472534179688, -0.6813411116600037, -0.07956735044717789, 0.485364705324173, 0.08777013421058655, 1.326675295829773, 0.9893448352813721, -0.07217276841402054, -0.004464369732886553, -0.4092566967010498, -1.8525595664978027, 1.0323172807693481, 1.0047763586044312, -1.9849234819412231, -0.9227136373519897, 0.5677347779273987, 0.4719891846179962, 0.9926613569259644, -2.7155771255493164, -0.08064932376146317, 0.3873532712459564, 0.5129826068878174, -1.2299567461013794, 0.5443320870399475, -0.15452012419700623, 0.01850976049900055, 0.13810285925865173, 1.40939462184906, -0.7987154722213745, -0.9090450406074524, -0.32021498680114746, 0.17931178212165833, -1.144350290298462, -0.3632267415523529, -0.159234881401062, 1.877639889717102, 0.6771473288536072, 2.1182329654693604, 0.22188133001327515, 0.6833962798118591, 0.09874226897954941, 0.4545857310295105, -1.4797836542129517, -0.6799288988113403, -0.02946566604077816, -0.7099409103393555, 0.2618354558944702, 1.908981442451477, 0.5406326651573181, -0.6833752393722534, -0.7575660347938538, 1.1105492115020752, -1.4434155225753784, -1.817503809928894, -1.2147773504257202, 1.8077378273010254, 1.5982961654663086, 0.10060343891382217, 0.25254151225090027, -0.8396726846694946, -0.6059549450874329, -1.0634660720825195, 0.7881917953491211, -1.5272631645202637, -0.6222953796386719, 0.9835540652275085, 1.152681589126587, 0.822236955165863, 0.2786402404308319, 0.11505718529224396, -1.5074000358581543, 1.1897804737091064, 1.586629867553711, -1.3898918628692627, -0.03326481953263283, 0.3453923761844635, -0.41211721301078796, -1.6008265018463135, 0.33600959181785583, -1.5377792119979858, -1.2293612957000732, -0.41089680790901184, -0.9444265961647034, 0.04949527233839035, 1.0601412057876587, 1.5005921125411987, -0.29761838912963867, -0.7234183549880981, 0.9236422777175903, 0.036233026534318924, 0.6151903867721558, -1.0017181634902954, 0.12800633907318115, 0.6138707399368286, 0.8901400566101074, 1.0520464181900024, 0.7853730916976929, 0.3851550221443176, 0.1073199063539505, -0.92789226770401, 0.43971529603004456, 0.7280213832855225, -0.5439230799674988, -0.3692886531352997, -2.2002174854278564, 0.5991092324256897, -1.4731537103652954, -0.24674749374389648, 0.006436101160943508, -0.727371335029602, -0.5278769135475159, -0.8266481757164001, -1.81206476688385, 0.18032145500183105, 0.7753077745437622, -0.8924121856689453, 0.22085189819335938, 1.5513757467269897, -1.1833043098449707, -1.7397143840789795, 0.9023537039756775, -0.9433687329292297, 1.200826644897461, 0.4192714989185333, -0.029066335409879684, -0.8479418158531189, -1.9945820569992065, 0.3901735544204712, -2.747303009033203, 0.023342646658420563, -0.19478559494018555, -0.046001650393009186, 0.36847323179244995, -0.13038715720176697, 0.5872008204460144, -0.7196944355964661, 0.2287992537021637, 0.5319129824638367, 1.2136690616607666, -1.7180825471878052, 1.168709635734558, -0.8463063836097717, -1.764701008796692, 0.20131051540374756, 0.6246442794799805, -0.015613229013979435, -1.6794970035552979, -0.6594356298446655, -0.6401455402374268, -0.8964779376983643, 0.40369871258735657, -0.10969875752925873, 0.1841534823179245, -0.9753303527832031, -0.23385556042194366, -0.5121268630027771, 0.5293617844581604, -0.45681124925613403, -0.8104987740516663, 0.7189521789550781, -1.7058157920837402, -1.2343525886535645, -0.6422502994537354, -0.33001741766929626, 0.5514531135559082, 1.9220613241195679, -0.780552327632904, 2.19224214553833, -0.3176012635231018, -0.8501399159431458, 0.29741236567497253, -0.44239985942840576, -2.5696260929107666, -1.2908066511154175, 0.8500859141349792, 1.03103506565094, 2.342170000076294, 0.5102884769439697, -0.869634747505188, -1.442280650138855, -1.3785291910171509, 0.9312995076179504, 0.6551998257637024, 0.40720459818840027, -1.7518075704574585, -0.4762265086174011, -0.06386756151914597, -0.8246473073959351, 0.8085524439811707, 0.3881838917732239, -0.0186587143689394, -0.6228896379470825, 1.4067530632019043, -1.061300277709961, 0.5890733599662781, 0.2960703670978546, 0.5736696124076843, -1.206568717956543, -0.1752540022134781, 0.48170605301856995, 0.37229812145233154, -0.004763427656143904, 1.2711036205291748, -1.019860029220581, 0.7873144149780273, 0.35297536849975586, -0.7054247260093689, -0.6911208033561707, 0.44045349955558777, -0.5299023389816284, 0.0330798365175724, 0.8298152089118958, -2.0798890590667725, 0.2588214576244354, 2.5816237926483154, -0.4076545834541321, -0.7124325633049011, -0.11056584864854813, 1.6999777555465698, -0.4953616261482239, 0.8239635229110718, 1.7340847253799438, 0.014684466645121574, 0.7412866353988647, 0.025035211816430092, 0.20870114862918854, 0.39134833216667175, 0.3914793133735657, -0.23214271664619446, -3.1950762271881104, -0.9432740807533264, 0.6680604815483093, 0.5402810573577881, -0.478502482175827, 1.4298604726791382, -0.9004641175270081, 1.0374433994293213, -1.0871516466140747, -0.33349889516830444, 1.0524332523345947, 0.06307636201381683, 0.3186268210411072, -1.4222135543823242, 1.925682544708252, -0.6544692516326904, -0.43810200691223145, 0.932348370552063, 1.3049238920211792, 0.4310095012187958, -1.4749224185943604, -0.032823774963617325, 1.3767393827438354, 1.196563720703125, -0.5638108253479004, 0.638050377368927, 0.040264792740345, 0.647083044052124, 0.21182605624198914, 0.04662192985415459, 0.9436610341072083, 0.9619098901748657, 0.9778484106063843, 1.5302685499191284, -2.1718642711639404, 0.9399729371070862, -0.4181993007659912, -0.13683082163333893, 0.9356498718261719, 0.402630090713501, -0.8054103851318359, -1.7539782524108887, 1.2000279426574707, -0.6899879574775696, 1.226431965827942, 1.0252087116241455, -0.319877028465271, 0.38691118359565735, 0.7346146702766418, -0.5746517777442932, 0.23750223219394684, 0.07474395632743835, -0.30422887206077576, 0.022059375420212746, -0.5514228940010071, -0.8909136056900024, -0.965815007686615, 0.4246140718460083, 0.03552534803748131, 1.1017696857452393, 0.8996540307998657, 1.8587216138839722, 0.798751175403595, 0.10676635056734085, -0.3108300566673279, -0.6347301602363586, 0.44198307394981384, 0.6950689554214478, -0.3483392000198364, -0.9719869494438171, -0.062037717550992966, 0.6889245510101318, 2.205082654953003, 0.12351828813552856, -0.9595500826835632, 0.10328519344329834, -0.938580334186554, -0.37781766057014465, 0.808385968208313, 1.912291169166565, -0.21567751467227936, 0.5981179475784302, 1.2507712841033936, -1.2448976039886475, -0.5591328144073486, -0.8527349829673767, -0.7649700045585632, 0.9841529130935669, 0.5577026605606079, -0.569232702255249, 0.4786105155944824, -0.28905564546585083, 1.7150659561157227, 0.5341489315032959, 0.4260544180870056, -0.22307495772838593, -1.5673257112503052, -1.1024434566497803, 0.132010817527771, -0.05018901824951172, 1.318359375, 1.4811803102493286, -0.5577464699745178, -0.08143686503171921, -0.6855612397193909, 0.46659931540489197, 0.33359721302986145, 0.8995648622512817, 0.09739591926336288, 0.7830842733383179, -2.378000497817993, 0.15445193648338318, 0.7476674914360046, -0.8421790599822998, -0.49016183614730835, -2.793039083480835, 0.3334815800189972, -0.2894972264766693, -0.45005619525909424, 1.6334357261657715, -0.7172420620918274, 0.08926640450954437, 0.30825182795524597, -0.955342710018158, 0.766758918762207, -0.012787227518856525, 0.6253748536109924, -1.539717435836792, 0.5806682109832764, 0.7668299078941345, -1.5397974252700806, -0.5133579969406128, -0.40859776735305786, -0.2867361307144165, 0.4299522340297699, -0.24635225534439087, 1.34586501121521, 0.34859412908554077, 0.8745065331459045, -0.6377950310707092, 0.8353424072265625, -0.7470306158065796, 1.7973670959472656, -2.132262706756592, -0.6733250617980957, 0.3121800124645233, -0.18977589905261993, 1.4884511232376099, 0.4571543037891388, -0.4941480755805969, 1.0185045003890991, 0.1084892526268959, -0.8327293395996094, -0.08524394035339355, 0.265516072511673, -0.22515642642974854, -0.2930455207824707, -0.3612861931324005, 1.0169506072998047, -0.5707434415817261, 0.6487061381340027, -0.4207246005535126, -0.7222832441329956, -2.184113025665283, -1.5026824474334717, 0.7028902173042297, 0.1018269807100296, 0.3736586570739746, -0.21946464478969574, -0.7942919135093689, 0.7307658195495605, 0.9248098731040955, -0.4585227072238922, -0.584869921207428, -0.45927539467811584, -1.1721727848052979, -0.5497007966041565, -0.37259194254875183, 0.023373417556285858, -0.19369523227214813, -0.8756551146507263, -0.9352115988731384, 0.8107677102088928, -0.3743760585784912, -1.13486909866333, 1.4694883823394775, -1.0053294897079468, -0.8147070407867432, -1.901066541671753, -0.6985989212989807, -1.1133551597595215, -0.597620964050293, 0.3739771842956543, 0.9557885527610779, 0.8512946367263794, 1.6172457933425903, -0.04632078483700752, 1.116914987564087, 1.4989979267120361, -0.6042065024375916, 0.29867470264434814, -1.639195442199707, 0.5495120286941528, -0.23012246191501617, 0.5699114799499512, -0.46485811471939087, 0.7225075364112854, 0.782425045967102, 0.14370940625667572, 1.016656517982483, -0.040744077414274216, 0.516549289226532, -0.8190768957138062, -0.5981358289718628, 2.1169614791870117, -1.23094642162323, -0.9459272623062134, -0.002946089021861553, -0.7583739757537842, -0.8119750022888184, -0.0026885047554969788, -0.22209197282791138, 0.7418854236602783, 1.399869441986084, 1.4762729406356812, 0.9649248123168945, 0.21872101724147797, -0.04672304540872574, 0.12621787190437317, -0.9144319295883179, -0.6238208413124084, 1.5589808225631714, -0.5144330859184265, 0.9044087529182434, 0.6950200796127319, -0.06782353669404984, 0.19338633120059967, -0.4059363305568695, 0.5477268099784851, -0.294428288936615, -1.7153388261795044, 0.2119351178407669, -0.7103487253189087, -0.4472736716270447, 0.790350615978241, -0.15090304613113403, 0.657854437828064, 0.9808515906333923, -0.4263627827167511, 0.01479710265994072, -1.1718686819076538, 0.05538787320256233, -1.2463864088058472, -0.8096529841423035, 1.5736860036849976, 0.6417968273162842, -0.9857960939407349, -0.8316925168037415, -0.6693857312202454, 1.4217467308044434, -0.5849601030349731, 1.4473602771759033, 0.8391783833503723, -1.503434181213379, 0.8908882141113281, 0.04627504572272301, 0.5997444987297058, -0.1232662945985794, -0.7174223065376282, -0.5572436451911926, 0.5046162605285645, -0.19477452337741852, -0.6512041687965393, 0.2748967111110687, 0.9014837741851807, -1.9079539775848389, -0.515098512172699, 1.446062445640564, -0.747326672077179, 0.9564217329025269, -0.272672176361084, -0.03239274024963379, -0.21345655620098114, 0.10189864784479141, -1.6659798622131348, 0.9954723119735718, -0.08730942755937576, 0.3492112159729004, 1.4629215002059937, -0.5750863552093506, -0.5926553606987, 0.6024826765060425, 0.6620029807090759, -0.7392056584358215, -0.5665601491928101, 1.8195276260375977, 0.7048893570899963, 0.10548306256532669, -1.5198025703430176, -0.9855319857597351, 0.8564859628677368, 1.8263297080993652, 0.3336560130119324, -0.4741036593914032, 0.03989628329873085, -1.0181376934051514, 1.0528641939163208, 1.1119120121002197, -0.4665273129940033, -0.5838784575462341, 2.0568292140960693, -0.43683722615242004, -0.26556870341300964, 1.1561347246170044, 1.0805391073226929, -0.08390256762504578, -0.9195430874824524, -1.4139113426208496, 1.3540055751800537, -0.12751248478889465, -0.784197986125946, -1.0033574104309082, -0.3228689730167389, 0.30583101511001587, -0.4212886095046997, -0.4023587107658386, -0.6776285767555237, -1.0601162910461426, 1.6318925619125366, 0.897815465927124, 0.8423488140106201, -1.6783630847930908, 0.3916487693786621, 1.5391740798950195, 0.025864411145448685, -1.1630319356918335, 0.32411515712738037, -1.08597993850708, -0.51138836145401, 0.8516446948051453, 1.106050968170166, 0.6203300356864929, -0.8704675436019897, -0.1019340306520462, 0.8307344317436218, 1.6156240701675415, -0.24298761785030365, -1.0263389348983765, 2.0978097915649414, 0.18712255358695984, -0.852963387966156, -0.3522062599658966, 0.1307086944580078, -2.517418622970581, 1.223551869392395, -2.0408546924591064, 0.3265596628189087, -0.2286820113658905, -3.0544493198394775, -0.4103338122367859, 0.7412866353988647, 0.9677406549453735, 0.28922122716903687, 0.20072317123413086, 1.2528316974639893, -0.5112575888633728, 0.4189717769622803, 0.2648380398750305, -0.381523072719574, -1.7267463207244873, -0.8469547629356384, 0.1305263489484787, 0.7533137202262878, -0.25633683800697327, -0.08970451354980469, 1.0220088958740234, 1.1068114042282104, -0.2520195245742798, 1.6438146829605103, -0.14404608309268951, 2.362102746963501, 0.11565178632736206, -0.12830036878585815, -1.2389634847640991, -0.7584447860717773, 0.060934245586395264, 0.016951803117990494, 0.4003170132637024, -0.6478689908981323, 1.2649624347686768, 0.518268883228302, -1.128087043762207, -1.4740315675735474, 0.15944506227970123, -1.4395145177841187, -1.5229928493499756, 0.2721810042858124, -1.884670615196228, 0.214456245303154, -1.3242794275283813, 0.04441377893090248, -0.945484459400177, -0.3422907590866089, -0.5219156742095947, -0.5569051504135132, -0.8175132274627686, -0.22490400075912476, 0.5907139778137207, -0.08373591303825378, -0.2860468029975891, 1.3705003261566162, -1.3015645742416382, 1.325752854347229, 1.3820245265960693, 0.7815054059028625, 0.4315721392631531, 0.8069342374801636, -2.051372528076172, -0.22714436054229736, 0.037661682814359665, -0.8165819048881531, 0.8914048075675964, 1.0579386949539185, 1.0337074995040894, -1.0013192892074585, 0.4917616844177246, 1.0615872144699097, 0.5131996273994446, 0.017691172659397125, 0.5013178586959839, -0.05588434636592865, 0.601897120475769, -1.4484293460845947, 0.15950186550617218, 0.9381818175315857, 0.33898937702178955, 0.4138946235179901, 0.63759845495224, 0.3460288643836975, 2.2038469314575195, -0.4261334538459778, 0.9261466264724731, 1.2149572372436523, -0.05377177894115448, 0.7979685664176941, -1.0971938371658325, -0.5932886600494385, 0.06426052749156952, 1.8638612031936646, 0.7740233540534973, 0.4929754137992859, 0.5432029366493225, 1.6162232160568237, 0.8461911082267761, -0.7396464943885803, 1.1392630338668823, -1.3814200162887573, -1.3130056858062744, -1.4278901815414429, -1.3710657358169556, -1.784078598022461, 0.4763326942920685, -0.2583538591861725, 2.500231981277466, -0.03935079649090767, -0.36944881081581116, -1.7734395265579224, 0.280356764793396, -1.2782553434371948, -1.2714529037475586, 0.4921610653400421, 0.5493025779724121, -0.14972436428070068, -0.2574610710144043, -0.6013249158859253, 0.5404438972473145, -0.3239763677120209, 0.4640502333641052, -0.2915551960468292, -0.2802434265613556, 1.2115402221679688, 0.21772165596485138, -1.5549286603927612, -1.254112720489502, -2.2007458209991455, 1.13681960105896, 0.48716872930526733, 0.250074565410614, -1.0728976726531982, 0.7534357905387878, -1.0125536918640137, -2.0247421264648438, -0.06100080534815788, 0.3266027867794037, -0.3702199161052704, 1.1737009286880493, -0.45622286200523376, 0.34507519006729126, 1.2616318464279175, 0.7030436992645264, -1.2121776342391968, 0.51691073179245, -1.118156909942627, -0.5388772487640381, -0.41960349678993225, -0.9728938341140747, 0.47137007117271423, -1.045553207397461, -1.1115506887435913, 0.9671305418014526, -0.8408936858177185, 0.049496252089738846, -0.5630056858062744, 0.8389931917190552, 0.7941111326217651, 1.253373622894287, -2.1791136264801025, -0.9879170656204224, -0.586681604385376, -1.3822021484375, -1.1019291877746582, 1.713276743888855, -1.5967917442321777, -0.6355782747268677, -0.5139021277427673, -1.0662444829940796, -0.6042613387107849, 0.8438758254051208, 0.5532553195953369, 1.6073508262634277, -0.982797384262085, 0.22754471004009247, -0.4365914762020111, 1.3145582675933838, 0.563057541847229, 1.6411699056625366, 0.6358166933059692, 1.2786246538162231, 0.0443841814994812, 0.1778108924627304, 0.5494518876075745, -0.4513314366340637, -2.0277645587921143, -0.9446457028388977, -1.5345884561538696, -0.07526066899299622, 1.7470369338989258, 0.5574744343757629, -1.2239443063735962, -0.3149296045303345, -0.014833072200417519, 2.9415597915649414, 0.2789020538330078, -0.43669605255126953, -1.2350205183029175, 0.1500423699617386, -0.744614839553833, -0.21694645285606384, -0.576154351234436, -0.34180039167404175, 0.5469942688941956, -0.09337043017148972, -0.5110902786254883, -0.2011609822511673, 0.3465835750102997, -0.7798718214035034, -0.12866295874118805, -2.2343215942382812, -0.23352324962615967, -0.6761899590492249, -0.24432769417762756, 1.4989221096038818, -0.04121985286474228, -1.2797248363494873, 0.09254051744937897, -0.1856568306684494, 1.2095930576324463, 0.6025643944740295, 1.3675079345703125, -1.0963555574417114, -0.7280997633934021, 2.0805420875549316, 0.3241448700428009, -0.04144873470067978, -0.6713457703590393, -0.4454081654548645, 0.2176007479429245, 1.2605072259902954, 0.0628509670495987, -0.12849238514900208, 1.3876056671142578, -0.028955576941370964, 2.144115686416626, -0.13376469910144806, 1.083858847618103, -1.813811182975769, 1.229462742805481, -0.18479673564434052, 1.2507892847061157, 1.6645798683166504, 1.038051962852478, -0.7956883311271667, 0.029465170577168465, 1.3375589847564697, 0.04534514993429184, -0.4879206717014313, 0.9092594981193542, 0.7643628120422363, -3.0820157527923584, 0.8525989651679993, -0.8662570118904114, -0.611552357673645, 0.6295458674430847, 0.9632163643836975, -3.0732719898223877, 2.460644245147705, -1.4114370346069336, -0.9824831485748291, -0.587598443031311, 1.847869873046875, 0.5869811773300171, 0.5354135036468506, 0.6225337982177734, 0.7629119753837585, -1.2910575866699219, 0.7114986777305603, -0.6832414269447327, -0.5752577781677246, 0.4446556270122528, -0.47233524918556213, 0.8386135697364807, 0.02615574188530445, 0.22847603261470795, -0.40050992369651794, 0.6623061895370483, -0.7524377703666687, 0.14666712284088135, 1.9538925886154175, -1.0668048858642578, -0.9717307686805725, -0.5258916020393372, 0.10207340866327286, -0.3799976408481598, 1.4132331609725952, 0.15162822604179382, -0.09782636165618896, 0.6092435717582703, 0.7018579244613647, -0.09554851800203323, 1.6384587287902832, -1.5578233003616333, -1.0801728963851929, -1.2260690927505493, -0.05512173846364021, 0.8766183853149414, 1.1866439580917358, 0.6016270518302917, 1.556368350982666, -2.0251779556274414, 1.4081605672836304, 0.1490412950515747, -0.0960737094283104, -0.32502037286758423, -0.7300891280174255, -0.6772269606590271, 0.20473666489124298, 1.0754014253616333, 1.9552319049835205, 1.4780012369155884, -0.11662072688341141, 0.19421853125095367, 0.1283116638660431, -0.4769986569881439, 1.1383373737335205, 0.08364947140216827, -0.7046542763710022, 1.6553707122802734, 2.0800538063049316, 0.5095487236976624, 1.7552731037139893, 0.8091944456100464, -1.438185214996338, 0.556596040725708, 3.0817620754241943, -0.06161881238222122, -2.4750149250030518, -0.8931957483291626, -0.7150745987892151, 0.17191757261753082, -0.49224022030830383, -0.40235915780067444, 1.2831230163574219, -0.4609531760215759, 1.9837090969085693, 0.06340517848730087, 2.327816963195801, 2.5532593727111816, -0.05957568436861038, -0.8802866339683533, -0.19739502668380737, -0.7283003926277161, -3.2348551750183105, -1.7729281187057495, 0.18530991673469543, 0.44504469633102417, 0.4182657301425934, -0.32285842299461365, 0.839460015296936, 0.1459040641784668, -0.22369223833084106, -1.6832079887390137, 0.5467193722724915, 1.2734484672546387, 0.18996642529964447, 1.6401718854904175, -0.12931233644485474, -0.390044242143631, 0.9927502870559692, -1.3563355207443237, 0.8484724760055542, -0.9967965483665466, 0.9315215349197388, -1.1582046747207642, -0.6718868017196655, -1.4735863208770752, 1.3981010913848877, -0.898612916469574, -1.88247811794281, 0.6678163409233093, -0.42544013261795044, -0.5962387919425964, -0.0980222076177597, -0.03253836929798126, 0.09577858448028564, 1.9581936597824097, -1.846369981765747, -1.074643611907959, 1.0255407094955444, -0.41536906361579895, -0.7330794930458069, 0.7464724183082581, -0.4466947615146637, -0.02095789648592472, 1.4491711854934692, -0.347881942987442, 0.24175383150577545, -0.8262366056442261, -0.7807548642158508, 1.5986286401748657, -0.4499265253543854, 0.9200740456581116, -0.6278311014175415, -1.3716315031051636, -0.27580249309539795, -0.002036250662058592, -0.35638710856437683, 1.335845947265625, 0.37944725155830383, -1.150380253791809, -0.649919867515564, 0.3291868567466736, 0.8028379082679749, 0.9684171080589294, 0.013658320531249046, 0.009298797696828842, -0.24035368859767914, 0.8468872904777527, 1.4466838836669922, 0.33335620164871216, 1.4199855327606201, 0.940597414970398, 0.9999895095825195, 1.6864206790924072, 0.18718776106834412, 0.5502689480781555, -0.6516930460929871, -0.7571895122528076, 0.795195460319519, 0.26527199149131775, 0.4944302439689636, 0.7394063472747803, 1.1926813125610352, 0.23741842806339264, 1.7375344038009644, -0.39431750774383545, 0.2122495174407959, 0.3262627124786377, -0.5354980826377869, -1.179659366607666, 0.19730144739151, 0.4778153598308563, 1.151059865951538, 0.4527692496776581, 0.10045584291219711, 1.152773380279541, -2.8625481128692627, 0.8686764240264893, 0.13318488001823425, 1.0087167024612427, -0.19681987166404724, 1.6068533658981323, -1.3558813333511353, -0.2952735722064972, 0.823884904384613, 0.2953198552131653, 0.3434467911720276, 0.9950178265571594, 1.1343662738800049, -1.555895447731018, -1.0419623851776123, 1.3074136972427368, -0.7647807598114014, 0.12134931236505508, 0.6499860286712646, 0.7302407026290894, 1.4996970891952515, 1.4591726064682007, -0.4332432746887207, -0.4424152672290802, 1.7352627515792847, 0.4253137707710266, -1.1078383922576904, -0.038914527744054794, -1.57035493850708, -1.1366673707962036, -0.9738389253616333, -1.9360531568527222, 0.5750423073768616, -0.278201162815094, -0.2128557562828064, 0.3993265628814697, -1.229364037513733, -0.8463471531867981, -2.2754383087158203, 0.6478231549263, -0.15301772952079773, 0.7310075163841248, 1.2287856340408325, 2.0392658710479736, -0.08839207887649536, -0.10393627732992172, 0.7347641587257385, -1.1731404066085815, -0.7182488441467285, -0.1547606736421585, 1.2343180179595947, 0.4329751431941986, 0.8838437795639038, -0.8094483017921448, 0.6002349257469177, -0.23800590634346008, 0.10624178498983383, -0.37281903624534607, 1.6171756982803345, 0.17825953662395477, 0.05308578908443451, -0.09275489300489426, 0.3249366283416748, -2.133267402648926, -0.9570409655570984, -0.40339159965515137, 0.1395314782857895, 0.5127807259559631, -1.3527418375015259, -0.237189382314682, 0.24665023386478424, 1.5087943077087402, -2.097820997238159, -1.1052831411361694, -0.06279700249433517, -1.3952269554138184, 1.8306013345718384, -0.09622929245233536, 0.21201089024543762, -0.7341010570526123, -0.018062887713313103, -1.2458842992782593, -1.910355806350708, -0.3388136327266693], "dtype": "float32"}, 1e-05], "e": {"__tup__": [{"__nd__": [[-1.3143452405929565, 0.3519771695137024, 0.31085366010665894, 1.1626255512237549, 0.3486228585243225, 1.2170120477676392, 0.011969095095992088, 0.4821648895740509, -1.1514582633972168, -0.21972638368606567, 0.2516911029815674, -1.5881760120391846, -0.17384199798107147, 0.045365069061517715, 0.1835278868675232, 0.6893500685691833, 0.7001144886016846, 0.19673757255077362, 0.10605312883853912, -0.9452052712440491, -2.1172542572021484, -0.22748936712741852, 0.05063697323203087, -0.8675948977470398, 0.32220780849456787, -0.23064930737018585, -0.33605751395225525, 0.5427655577659607, 0.19824005663394928, 0.08193962275981903, -1.1419904232025146, -0.26963868737220764, 0.40027710795402527, -0.5514718890190125, -0.02445932850241661, 0.9372872710227966, -0.2587619125843048, -0.11602305620908737, -0.6770455241203308, -0.1948760747909546, -0.20816251635551453, 1.3882273435592651, 0.39275702834129333, 0.0018621687777340412, 4.635841469280422e-05, -0.3598710298538208, 0.12637552618980408, 0.6679365038871765, -1.8191332817077637, -0.5900209546089172, 0.14013606309890747, -0.05465851351618767, -0.058036837726831436, -0.9427356719970703, 0.0876852422952652, 1.0141055583953857, 0.023266490548849106, -0.008168146014213562, -0.16464035212993622, -0.0947560966014862, 0.353263258934021, -0.1631731241941452, -0.25972384214401245, -0.623587429523468, -1.6790881156921387, 0.13308005034923553, -0.9715921878814697, -1.68245530128479, -0.00892659556120634, -0.7638182044029236, 0.05336732789874077, -1.3803203105926514, 1.3527944087982178, -0.42396360635757446, -0.5897424817085266, -0.15755215287208557, 0.00012470553338062018, -2.4376943111419678, 0.04530873894691467, 0.16039632260799408, -2.7540500164031982, 0.6857221722602844, 0.026041192933917046, -0.07971253991127014, 0.2042698711156845, -0.4872726798057556, 0.08953354507684708, 0.18040473759174347, -0.362348347902298, 0.5011321902275085, 0.5356485843658447, -0.07539968192577362, -0.03348173573613167, 0.059802908450365067, -0.5097404718399048, -0.2043578177690506, 0.2585296332836151, 0.37767601013183594, 1.7567647695541382, 0.09338237345218658, -0.03418481722474098, 1.3242517709732056, 1.474595069885254, -0.05320623889565468, -0.11679258942604065, 0.006543494760990143, -0.5867694616317749, -0.39698490500450134, -0.39373525977134705, -0.3577640950679779, -1.2925221920013428, 0.014312484301626682, 0.15773873031139374, 0.22798432409763336, -0.39779138565063477, -0.5303886532783508, 1.845511794090271, -0.926734209060669, 1.2155673503875732, -0.8213819861412048, 0.056133825331926346, -0.2985159754753113, 0.10414112359285355, -1.5899231433868408, -0.06168041378259659, 0.06556708365678787, -0.20894256234169006, 0.0111932884901762, 1.3728734254837036, 0.6035122275352478, -0.4074784219264984, 1.3654814958572388, -0.08596239984035492, -0.0833262950181961, -0.8826279640197754, 1.022625207901001, -0.06942198425531387, 0.4220329225063324, 0.870510995388031, 0.10793135315179825, 0.08479254692792892, 0.05241142958402634, 0.24728482961654663, 0.06157607585191727, 0.3862815499305725, 0.6158950328826904, 1.5617294311523438, -0.7631677985191345, -0.026183389127254486, -0.06301924586296082, 1.8499212265014648, 0.4876236915588379, -0.20673798024654388, 0.3864057958126068, 0.7936446666717529, -0.4693197011947632, -0.4359202980995178, -0.2462136149406433, -0.3556775748729706, -0.6482672691345215, -0.9214330315589905, -0.040036216378211975, -0.0019095346797257662, 0.6632590889930725, -0.5816264152526855, -0.04776174947619438, -0.3873339891433716, 0.5169246792793274, 0.4441222846508026, -0.11080313473939896, 1.089339017868042, -0.35638347268104553, 1.9050695896148682, 0.4963362514972687, 0.21672789752483368, -0.18906527757644653, -0.026593856513500214, 0.498696893453598, -0.2727738320827484, -0.6216882467269897, 0.7552412748336792, 0.007773874327540398, -0.03903598710894585, -0.39586034417152405, -2.064084529876709, 0.06674746423959732, -0.7797098755836487, -0.002951474627479911, -0.8819288015365601, 0.694756031036377, 0.0033537682611495256, -0.12767332792282104, 0.5703966617584229, 1.9213298559188843, -0.6674400568008423, -0.22672942280769348, 0.464664489030838, -0.3728342354297638, 0.6826880574226379, 0.5954650044441223, 0.13568516075611115, -0.010634709149599075, -0.05581388995051384, 0.1688069850206375, 0.21437467634677887, 0.4210977256298065, 0.6038891673088074, -1.938330888748169, -0.8311592936515808, -0.008337045088410378, -0.003779234364628792, 0.5290772914886475, -0.0653172954916954, -0.014943236485123634, -0.27949631214141846, 0.7032207250595093, 2.4332199096679688, -0.21418875455856323, -0.030967462807893753, 0.1480005532503128, 0.6505457162857056, 0.30758723616600037, 0.2765970230102539, 0.240992933511734, -0.24889235198497772, -0.08532685786485672, 1.1668392419815063, 2.5299715995788574, -0.10196909308433533, 0.09403658658266068, 0.06912952661514282, -0.21470899879932404, 2.5986361503601074, 0.8660709857940674, -1.3720611333847046, -0.9064339995384216, -0.2651158273220062, 0.5334445238113403, 0.21695147454738617, 0.031246956437826157, -0.9628520011901855, -0.23943060636520386, -1.6152316331863403, 0.8738571405410767, -0.7854554653167725, -0.02599320001900196, 1.4469293355941772, -1.2085793018341064, 0.0643390640616417, -0.12434118986129761, -0.0717829167842865, 0.8618617653846741, -0.18271450698375702, 0.016484558582305908, 1.932167410850525, -0.1253972202539444, 0.6038667559623718, 0.14778394997119904, -0.10973833501338959, 1.5321862697601318, 0.7766984701156616, -0.8713641166687012, -0.24773278832435608, -0.6784878373146057, 0.3438904285430908, -0.4618734121322632, 2.133460760116577, 0.16167916357517242, 1.1631602048873901, -1.1270633935928345, -0.0005755940219387412, -0.21526646614074707, 0.15978309512138367, 0.2268926054239273, -0.08822574466466904, -0.010893172584474087, 0.053055860102176666, -0.08779498934745789, 0.21120083332061768, 0.8958439826965332, -0.11722040176391602, 0.162658229470253, 0.8076114058494568, 0.47479167580604553, 0.5715910792350769, 0.3081701993942261, 0.44478917121887207, 1.2740517854690552, 0.24458377063274384, -0.8334237337112427, -0.2884546220302582, -0.5638839602470398, 4.169836044311523, 0.26426130533218384, 0.1839071363210678, 0.4351366460323334, -0.17698934674263, 0.018750295042991638, 1.0416682958602905, 0.073421910405159, -0.32680168747901917, -0.04801614210009575, 0.14696137607097626, -0.9793057441711426, 0.26912766695022583, 2.076078176498413, -0.025646701455116272, 4.69124698638916, -0.22113312780857086, 0.11855985969305038, -0.016920577734708786, 1.095747470855713, 0.02158018760383129, -0.6562512516975403, 2.3905029296875, 0.5715686678886414, 1.4759135246276855, -0.10846663266420364, 1.1290034055709839, 0.08874400705099106, -0.4801422953605652, 0.2537490725517273, -0.08474498987197876, 0.9834545254707336, -0.03634146228432655, -0.32287073135375977, 0.4436156749725342, -0.8891812562942505, -0.3049159646034241, -0.347289115190506, 0.3445650041103363, -0.36779242753982544, -0.09969063848257065, -0.006341940723359585, 0.3837302029132843, 0.6584861874580383, -2.362173557281494, -1.7788714170455933, -0.23775731027126312, 0.1424979269504547, -0.6040685772895813, 2.1583893299102783, 0.8191718459129333, 0.897102415561676, 1.1188052892684937, -1.8019452095031738, -0.5511574149131775, 0.00815381109714508, -1.1461211442947388, -0.023772479966282845, 0.6625978946685791, -0.7352568507194519, -2.5573179721832275, 0.8989123106002808, -0.295428991317749, -0.5898249745368958, -0.19352082908153534, 0.5762118697166443, -6.261497497558594, 1.203994870185852, 0.09131525456905365, -0.25958052277565, 1.3016854524612427, 0.1152016744017601, -1.1249277591705322, -0.1667056679725647, 0.43091192841529846, 0.6537740230560303, 0.047147221863269806, 1.1848334074020386, -0.09179811179637909, 0.021709982305765152, 0.09479972720146179, 0.6179535388946533, -0.5459056496620178, -0.8618915677070618, -0.0919584333896637, 0.18144741654396057, -0.7074867486953735, 0.39413511753082275, 0.2148606777191162, -0.18414436280727386, 0.009747886098921299, 2.504246234893799, -0.7603799104690552, -0.5285881757736206, 0.01434866152703762, -0.0922205001115799, -1.1869022846221924, 0.278049111366272, -0.7554888129234314, -0.18142913281917572, -0.12843458354473114, -0.04901016131043434, -0.2786547541618347, -0.12266912311315536, 0.12684401869773865, -0.0799950659275055, -0.14863169193267822, 0.0002477098605595529, -0.6707324981689453, 0.6761157512664795, -0.9794309735298157, 0.39647844433784485, -0.014101231470704079, -1.9964702129364014, -2.0811994075775146, 0.10266728699207306, -1.7511264085769653, -0.8810838460922241, -0.10182041674852371, -1.0261527299880981, -1.062975287437439, 2.178814172744751, 0.06386806815862656, -0.31524658203125, -0.032611168920993805, 0.29070332646369934, 0.5893200039863586, -0.4152764678001404, 0.09861762076616287, 1.465132474899292, -0.23378179967403412, 0.18443533778190613, 0.5376782417297363, 2.8254406452178955, -0.030843185260891914, 0.3026371896266937, 0.13586433231830597, -0.18862606585025787, -0.9997922778129578, -0.07470404356718063, -0.42737874388694763, 0.27659836411476135, 0.31016451120376587, 0.42482009530067444, -0.13473792374134064, 0.02382078766822815, 0.3969956636428833, 1.7676670551300049, 1.3368419408798218, 0.2037285715341568, -0.48274901509284973, 0.0441422276198864, -0.3043581545352936, -0.508582592010498, 2.59898042678833, 1.1809124946594238, 1.1248663663864136, 0.005735353101044893, -1.7257237434387207, 1.0185680389404297, 0.003336624475196004, 0.42552411556243896, -0.6152607202529907, 0.7217143177986145, 0.09634291380643845, -1.1341893672943115, 0.8516741394996643, 0.0033867117017507553, -2.0530574321746826, -0.12703177332878113, 0.03838077187538147, 0.1473993957042694, -0.1523859202861786, 0.4441259503364563, 1.3229649066925049, -0.06696140021085739, 0.40878623723983765, 0.5000925660133362, 0.04035568982362747, 0.6601451635360718, -0.7280321717262268, -2.139470338821411, 0.3263612389564514, -0.28134769201278687, 0.961421012878418, 0.5949025750160217, -0.38579627871513367, -0.9137723445892334, -0.04082794860005379, -0.2838613986968994, -0.18770743906497955, 2.3952932357788086, 0.7747787237167358, 0.32858189940452576, 0.6337369680404663, -0.6586377620697021, 0.78436678647995, 0.005937937181442976, 0.628147304058075, -0.9712129235267639, -0.015455149114131927, 0.9272500872612, -0.002412236761301756, -0.31381896138191223, -0.6875135898590088, -0.3209090828895569, -1.897039771080017, -1.2866592407226562, -6.49045467376709, -2.3805651664733887, 0.4302195608615875, 0.007782360073179007, 0.2069777101278305, -0.05536508187651634, 0.35512775182724, 0.21503151953220367, 0.26262858510017395, -0.9362619519233704, -0.3204321265220642, 0.05324533209204674, 0.021890494972467422, 0.7299636602401733, 1.173317551612854, 0.3163406252861023, 0.7906951904296875, 1.9685505628585815, -0.8232864141464233, -1.1561384201049805, 0.29099297523498535, 0.24651166796684265, -0.7267392873764038, -1.4079312086105347, -1.3370060920715332, 0.3573327362537384, -0.07245195657014847, -0.7025142908096313, 0.2595970332622528, -0.5918300747871399, -0.09218984097242355, 0.5532031059265137, -0.21880970895290375, 0.08483020216226578, 0.8661317825317383, 0.7081777453422546, -0.46943357586860657, 0.6559742093086243, -1.1741728782653809, 1.8516230583190918, -0.21219223737716675, 0.005068029277026653, -1.6779022216796875, 0.04612978547811508, -0.8652791380882263, -0.0030395686626434326, -0.7109953761100769, 0.1555938869714737, -0.1300847977399826, 0.010938878171145916, -0.0193694569170475, -0.11844673752784729, -0.7633375525474548, -0.1764642894268036, 0.7189527153968811, 1.3785573244094849, -0.812546968460083, 0.4930005669593811, 0.02115144394338131, 0.021521715447306633, 1.6292818784713745, 0.7126550674438477, 0.032291948795318604, 0.25552600622177124, 1.1120803356170654, 1.9402990341186523, -0.008840087801218033, 0.21154843270778656, -3.800961494445801, 0.26966193318367004, 0.7829142212867737, 1.20504891872406, 0.5686710476875305, -1.6988939046859741, -0.6466153860092163, 0.3558296859264374, -4.248185634613037, -0.3074452877044678, 1.2824310064315796, 0.10329796373844147, 1.4590084552764893, -0.3453690707683563, 0.10534238815307617, -0.014326818287372589, -0.04447537660598755, 2.5752060413360596, 0.17008572816848755, -0.035245321691036224, 0.7886715531349182, -0.1789642572402954, -0.5597751140594482, -0.30468326807022095, -0.19141411781311035, 0.2014402598142624, -0.004704353399574757, -0.012736175209283829, -2.094514846801758, -0.1832457035779953, 2.5719711780548096, -0.055238135159015656, -0.2620903253555298, -0.29368695616722107, 0.20781345665454865, 1.0351359844207764, -0.004019045736640692, 1.0159493684768677, -0.0043297722004354, 0.7218886613845825, 0.6005004644393921, -0.7492902278900146, -2.486757755279541, -0.059263307601213455, 1.53121018409729, 0.19393686950206757, 0.16805021464824677, -0.046954743564128876, -0.10249306261539459, 0.7112379670143127, 0.3068765103816986, 0.45003461837768555, 0.0062960064969956875, 0.10548439621925354, 0.501690685749054, 3.8958253860473633, -0.18755806982517242, -0.7010255455970764, 2.1213626861572266, -0.23215116560459137, -0.38009682297706604, 0.2331314980983734, 0.2690350413322449, 0.10978220403194427, 0.0066931117326021194, -0.6673921942710876, -0.900273859500885, 0.13310863077640533, 0.20454642176628113, -1.0972555875778198, 0.06287437677383423, -1.4622234106063843, 0.660885751247406, -0.10692787170410156, 0.2550275921821594, 1.230876088142395, -0.05170834809541702, -2.287881374359131, -1.1641604900360107, 0.10364368557929993, -0.3286516070365906, 0.04659195616841316, 0.9709658622741699, 0.1736462563276291, 0.013431853614747524, 1.523569941520691, -0.6605900526046753, -0.014875777065753937, 0.035152725875377655, -0.4036915600299835, 0.020051000639796257, 1.150959849357605, 0.07730744779109955, -0.03326563909649849, -0.04235169664025307, 0.04962698742747307, -3.483191728591919, 0.09066317230463028, -1.5938720703125, -0.39417892694473267, 0.28306373953819275, 0.03339449316263199, -0.05067317187786102, -0.1493207961320877, 1.212086796760559, -0.17599262297153473, -0.8491731286048889, 2.227956771850586, 0.4809817373752594, 0.048805106431245804, -0.6222293376922607, 0.19123585522174835, -0.04263510927557945, 0.02443704567849636, 0.00017436551570426673, 0.17212866246700287, -0.023755881935358047, 0.15792006254196167, 0.013025572523474693, 0.1180267333984375, 3.0634381771087646, -0.6439399123191833, -2.3068041801452637, 0.391665518283844, -0.4705645442008972, 0.23305103182792664, 0.5783019065856934, 0.2363961935043335, -0.34262892603874207, 0.0010213032364845276, 0.03030339628458023, 3.299640655517578, 0.0029821177013218403, 0.7306011915206909, -0.3837522864341736, 0.10500944405794144, -0.5451396703720093, 1.7873649597167969, 0.09730624407529831, 0.022374089807271957, 2.0060312747955322, 3.0206987857818604, -0.825222909450531, 0.4584091305732727, 0.6155185103416443, -0.45212522149086, -0.07894432544708252, -3.0404977798461914, 0.2644864618778229, -0.07610076665878296, -1.127109169960022, -0.6172188520431519, -1.8166708946228027, -0.05844418704509735, 0.04784700274467468, 0.756453275680542, 1.5338739156723022, -0.7750095129013062, 0.03689464181661606, -0.8244567513465881, -0.42950206995010376, 1.0842663049697876, -0.6308870315551758, 0.2700093686580658, 0.48299482464790344, -0.023022830486297607, -1.226446509361267, 1.1385972499847412, 0.06024356186389923, 0.07602886110544205, -0.0021297908388078213, -0.38396647572517395, -0.0017412632005289197, 0.1754794418811798, -2.749340295791626, -0.10947208851575851, -0.036891452968120575, 0.40079280734062195, -0.0004915173049084842, 0.09368932247161865, 0.40598028898239136, -0.12140796333551407, 0.8210569024085999, -1.1337889432907104, 1.7984919548034668, -0.37485599517822266, 0.6295566558837891, -0.05997694283723831, 0.3290438950061798, -0.16906051337718964, 0.3861294090747833, 0.027233213186264038, -1.0663976669311523, 0.12015999108552933, -0.31638845801353455, 0.22340022027492523, 0.44261598587036133, 0.4614945352077484, 0.38195809721946716, 0.11373069882392883, -0.08952204883098602, 0.3395732641220093, 0.0003579258918762207, -3.219151735305786, 0.8753024339675903, 0.05739356949925423, 0.0013379387091845274, -0.07116281241178513, 0.9791313409805298, -0.6402295231819153, -0.20655310153961182, -1.6283013820648193, -0.015671027824282646, -2.131359100341797, -0.41314125061035156, -0.04450371488928795, 1.2916538715362549, 0.010164516046643257, 0.5293609499931335, -0.7812584042549133, -0.2170013189315796, 1.3612991571426392, -0.3176302909851074, -0.623418927192688, 0.25788378715515137, -0.2132958322763443, -0.001111313235014677, 0.2934551239013672, -0.17248545587062836, 0.21502017974853516, -0.23517683148384094, -0.08066453039646149, 1.196662187576294, 0.1073589101433754, -0.034570544958114624, 0.19019505381584167, -3.2982099056243896, 0.11721006035804749, -0.484171062707901, -0.4568711221218109, -0.9499831199645996, -1.628994345664978, 0.468570351600647, 0.05251824110746384, 0.7160067558288574, 0.043334703892469406, -0.09556997567415237, -0.07659027725458145, -0.032009877264499664, -0.04909573495388031, 1.8695307970046997, 0.7763553261756897, -0.38217440247535706, 0.1877967119216919, 0.19176623225212097, -0.0018753288313746452, 3.089132070541382, -0.1267625391483307, -0.01248922385275364, -0.026227876543998718, 0.43242865800857544, 0.7260029315948486, -0.6046114563941956, -0.004604310728609562, -0.9730501174926758, 0.14896811544895172, -1.9774061441421509, 1.9469975233078003, 0.05403384193778038, -0.9635055661201477, -0.048407383263111115, 0.9913877844810486, -2.174999713897705, -1.404901385307312, -0.06304225325584412, 1.2813467979431152, -0.9455108046531677, 0.7814124226570129, 0.5795722007751465, -0.7969886064529419, 0.03650263696908951, 0.13795872032642365, 0.2006673961877823, -0.04719175770878792, 0.6369989514350891, 0.23429778218269348, 0.4050697386264801, -1.1951894760131836, -0.11820998042821884, 0.30959609150886536, 0.0021821539849042892, -0.7884966135025024, 0.20532049238681793, 0.4690936207771301, -1.6219278573989868, -0.05808177590370178, -0.38440704345703125, 0.007255746517330408, 0.22295542061328888, -0.008296169340610504, -0.5939602255821228, 0.3879057765007019, -0.8159694075584412, -0.27069583535194397, 0.6113003492355347, 0.003891457337886095, -0.653685450553894, -1.9445666074752808, -0.028881285339593887, -1.5492732524871826, -0.03909355029463768, -0.09036996215581894, 0.8698571920394897, 0.8887984752655029, -1.0332452058792114, -0.19123932719230652, 0.27622735500335693, -0.02680918201804161, 0.20757147669792175, -0.2264968603849411, -0.08124598115682602, 0.3134741485118866, -0.1950879991054535, -0.9036356210708618, 0.014461376704275608, -0.6926202774047852, 0.07964310795068741, -1.0027761459350586, 1.6254339218139648, 0.29121556878089905, -0.24301990866661072, 2.7436232566833496, -0.7008334994316101, 0.4425380825996399, -1.3499584197998047, 0.0031385605689138174, -2.529146432876587, -0.0379909984767437, 0.062154676765203476, -0.5373342037200928, 0.5409795641899109, 0.5526810884475708, 1.5265082120895386, 0.9667619466781616, 1.4216591119766235, -0.5519024729728699, 0.4723852574825287, -0.7236106395721436, -1.5274149179458618, -2.1877286434173584, 0.02072630263864994, 1.4370174407958984, -0.3451177775859833, -1.8755404949188232, -0.6869455575942993, -0.004011623095721006, -1.86854088306427, -0.37093499302864075, -1.0642000436782837, -0.5304636359214783, 0.08863961696624756, -0.12901560962200165, -0.3704139292240143, 0.8944376111030579, 0.40834537148475647, 1.470701813697815, 0.1302344799041748, -0.23007583618164062, 0.617518424987793, 0.39746302366256714, 0.8880655169487, -1.2908331155776978, -1.1110111474990845, -0.33674347400665283, -0.4010162949562073, 2.475773334503174, -0.38981136679649353, 0.01752629317343235, -0.23113052546977997, 1.4544025659561157, -0.2767842411994934, 0.11204621940851212, -1.0655220746994019, -0.3405616283416748, -0.1260891854763031, 0.3039851486682892, 0.585697591304779, -0.27011948823928833, -0.4239148199558258, -1.305097222328186, 1.3009071350097656, -0.13929781317710876, -4.47525691986084, -1.2799245119094849, 6.121149063110352, 0.6787792444229126, 0.6812630891799927, 0.2741217017173767, 0.4589075446128845, -0.04405852407217026, 0.42947736382484436, 1.8593217134475708, 0.1553397923707962, -0.3873666822910309, -0.09694782644510269, 0.00442378269508481, 0.99186772108078, 0.17641451954841614, -0.2927321195602417, -0.09547226130962372, -5.565696716308594, -0.01393651869148016, -1.0826067924499512, 0.5196933746337891, -0.4400371015071869, 0.018324311822652817, 0.40402716398239136, 0.08636221289634705, 1.3539104461669922, -0.0392027385532856, 0.012763273902237415, -1.524381399154663, 0.008716280572116375, -1.3847366571426392, -0.12951931357383728, 0.992992103099823, 0.00035206571919843554, -0.6659637093544006, -1.9763869047164917, 0.6853747963905334, 0.49956759810447693, 0.03062969446182251, 1.8391975164413452, 2.8012890815734863, -0.8035029768943787, -1.690150499343872, -0.050715960562229156, 0.2896284759044647, -0.1468435674905777, 0.15978968143463135, -0.09182532131671906, 0.6668531894683838, 0.04496016725897789, -0.2874050736427307, -0.3707393407821655, 1.009459137916565, 0.7674898505210876, -1.9484219551086426, -0.17995722591876984, 0.489313006401062, 0.07502751797437668, -0.7110615372657776, 3.902301549911499, 0.46762362122535706, 0.15186484158039093, 0.00030893125222064555, -0.23631493747234344, -0.14091886579990387, 1.1316673755645752, 0.4818527400493622, -0.01695477031171322, -0.13410605490207672, -0.36714914441108704, -0.5744215250015259, 1.7830730676651, 0.006706012412905693, -1.0858074426651, -0.4079625606536865, 0.3527810275554657, -0.9486139416694641, -1.0977039337158203, -0.02297823317348957, 0.2643251121044159, -0.3267545998096466, 0.002469110768288374, -2.0637826919555664, 0.3840555250644684, 0.613797664642334, -0.4341263473033905, 0.0942080169916153, 0.42176932096481323, -3.1696739196777344, -0.3147006928920746, -0.6786043047904968, -0.2767293453216553, -1.796557068824768, 0.12861788272857666, 2.847456932067871, 1.3482346534729004, 1.9014891386032104, -0.5958261489868164, -0.32166972756385803, 0.32350313663482666, -0.41827940940856934, -0.18584468960762024, 0.192804753780365, 0.22908958792686462, -0.0404709167778492, -0.02368278056383133, 0.7432342767715454, -1.8408340215682983, 0.06075828894972801, 0.31990474462509155, 0.11562982946634293, -1.0538675785064697, 5.5554118156433105, -0.7510771751403809, 0.0014419410144910216, -0.5473383665084839, -0.49772533774375916, -3.028247594833374, 0.4006868004798889, 1.324763298034668, 1.3495601415634155, 0.07186438888311386, 0.12538306415081024, -0.25856339931488037, -0.07598206400871277, -0.4076314866542816, 0.015402429737150669, -0.26380273699760437, -0.39783498644828796, 0.7565013766288757, -0.33093366026878357, -0.12139546871185303, -0.9552677273750305, -0.003403163980692625, 0.8787025809288025, 0.7907404899597168, -0.03843167796730995, 0.33311012387275696, 0.07843481004238129, 2.5667877197265625, 0.36097079515457153, -1.3408241271972656, -0.4130478501319885, -0.18947875499725342, -0.8114036321640015, 0.3404143750667572, 0.05673902481794357, 1.3727375268936157, -2.469019889831543, -0.8916771411895752, -0.07027864456176758, 0.35728514194488525, -0.11984732747077942, -2.301424264907837, 0.788359522819519, -0.012989974580705166, -1.623325228691101, 0.6852277517318726, -0.1744995415210724, -0.1947975903749466, 0.5262325406074524, 1.8853275775909424, 0.4347110688686371, -1.9542802572250366, -3.7953364849090576, 0.5957005620002747, -0.3276689946651459, 0.11660008877515793, 0.44759833812713623, 0.07859596610069275, 0.02959837205708027, -0.26135265827178955, 0.49101611971855164, -2.695955753326416, 0.2617489695549011, 0.5824790000915527, 0.05854782834649086, 0.45777642726898193, 0.9947777986526489, 0.7307798862457275, -1.9083305597305298, 6.253622996155173e-05, -0.3592849373817444, -0.13052265346050262, 0.1280786544084549, -0.01601761393249035, -0.06233436241745949, 0.5158190131187439, -1.188024878501892, 0.3177936375141144, 0.3954368233680725, -0.24161268770694733, -0.05543550103902817, 1.0048003196716309, -0.5889766812324524, 2.136005163192749, 0.0864877998828888, 0.33170363306999207, -0.8922260999679565, 0.4675835967063904, 0.3820425271987915, 0.7239003777503967, 0.285551518201828, -0.6102158427238464, -0.8529481887817383, 3.869532585144043, 2.5695087909698486, -0.0968031957745552, -0.04565073549747467, -0.15916720032691956, 0.5772385597229004, -1.2887284755706787, -0.023776229470968246, -0.6030867099761963, 0.28421708941459656, 0.08223695307970047, 0.2628509998321533, 0.5240796208381653, -0.7940940260887146, -0.3728734254837036, -0.003930169157683849, -0.018732089549303055, 0.08884575963020325, 0.028870608657598495, -2.0346357822418213, 0.10285483300685883, 0.05978715792298317, 0.004682967439293861, -0.05031908303499222, -0.2188967913389206, -0.01862240582704544, -4.451998233795166, 0.003260466968640685, 0.3148752748966217, 0.25822874903678894, -0.0988389179110527, -1.4648808240890503, -0.05358922481536865, -0.26837828755378723, -1.0346863269805908, 0.8698083162307739, 0.8471750617027283, -0.3121633529663086, -0.061656732112169266, 0.10290451347827911, -0.22052082419395447, -0.07952868193387985, 0.5008335113525391, -0.07496239989995956, -0.047861117869615555, -0.5423871874809265, -0.09279754012823105, -0.89734947681427, -0.22942957282066345, -0.7465474605560303, 0.2068510353565216, 0.06055152416229248, 0.24163466691970825, -0.05250391364097595, 9.35818170546554e-05, 0.21267500519752502, -0.8451893329620361, 0.41885900497436523, 0.1708277463912964, -0.8178114891052246, -0.6339555978775024, -0.052496932446956635, -0.023068463429808617, 0.5792966485023499, -0.14670422673225403, -1.1190425157546997, -0.4283994138240814, 0.9389559626579285, -0.21448177099227905, -2.0340147018432617, 0.10038397461175919, 0.04410172253847122, 0.3041655719280243, -0.33730655908584595, -2.197206974029541, -0.183163121342659, 0.6503592729568481, 0.5622684359550476, 0.7006000280380249, -0.014565713703632355, 0.16767020523548126, -1.6957279443740845, -0.054772309958934784, -0.44510746002197266, -0.021210072562098503, -1.8491690158843994, 0.7979194521903992, 1.242291808128357, 0.6151360273361206, 0.6518079042434692, 0.2727852463722229, 0.49167853593826294, 0.4085617661476135, -0.3854835331439972, -0.03947895020246506, 0.25869011878967285, -0.2606026530265808, -1.2469863891601562, -0.253233939409256, -0.5747572183609009, -0.3119879364967346, 0.44535335898399353, -0.29794830083847046, -0.002799222944304347, 0.25043776631355286, 1.675792932510376, -0.022011741995811462, 1.4407697916030884, -0.5264238119125366, 0.8377954959869385, -0.9666202664375305, -0.005730051547288895, -0.8735566735267639, 0.47647643089294434, -0.300402969121933, -0.7955987453460693, -1.1445404291152954, -0.040931105613708496, -0.013174980878829956, -0.1552020013332367, 0.3563990592956543, -0.5980880260467529, -0.08543034642934799, -1.4595123529434204, 1.0778846740722656, 0.011638741008937359, 0.09286051988601685, 0.15119197964668274, -0.05484894663095474, 0.5124725699424744, -0.3687819838523865, 0.20589102804660797, 1.0283969640731812, -0.44434601068496704, -0.11022929102182388, 0.08656047284603119, -0.10525520890951157, 1.5806355476379395, 0.13648876547813416, 0.017600512132048607, 1.3566985130310059, -0.36902916431427, -0.4165167212486267, -1.1212396621704102, -0.03188121318817139, 0.005096169654279947, -0.8808527588844299, 0.9551708698272705, 0.05622832104563713, 0.08541592210531235, -0.15629909932613373, 0.06556664407253265, 1.6113239526748657, 0.2017630636692047, 0.035140059888362885, -0.40208932757377625, -0.398003488779068, -0.4701937437057495, 2.008188247680664, 0.9625667333602905, -1.6630640029907227, -0.11446257680654526, -0.125653937458992, -1.1147971153259277, 0.3419933617115021, -0.5268452763557434, -0.0058474186807870865, 0.3160138726234436, 0.10123521834611893, -1.2794008255004883, -0.21879014372825623, -0.8133919835090637, 0.04008081182837486, 2.655867338180542, 0.002026574919000268, -0.6151876449584961, -0.40813755989074707, 0.06188677251338959, -0.4101296663284302, -0.2546093165874481, 0.09805551916360855, -1.1131799221038818, -1.7117668390274048, -0.43704667687416077, -0.8756328821182251, 0.4797356128692627, -0.9195737838745117, 0.27379950881004333, 0.5548151731491089, -0.2613987326622009, -1.9936190843582153, -0.14860717952251434, -0.36667492985725403, -1.264565110206604, 0.10203637182712555, 0.46910637617111206, -0.06449197232723236, 1.938799262046814, -0.7712830901145935, -0.602272093296051, 1.1373255252838135, -0.06544730067253113, 0.005122383125126362, -0.008766436949372292, 0.04043612629175186, 0.1499934047460556, 1.0734045505523682, -0.03270222246646881, -0.7911484837532043, 0.7147148251533508, -0.20446895062923431, 0.24009297788143158, -0.6115638017654419, 0.03362392261624336, -0.032061878591775894, 0.5181963443756104, 0.03975531458854675, -0.25456392765045166, -0.10970952361822128, -3.4509029388427734, -0.716391384601593, -1.5646324157714844, 1.7293134927749634, 1.2722811698913574, -0.19433988630771637, 0.18808552622795105, 0.22035223245620728, -0.09813833236694336, -0.8735974431037903, 1.4326939582824707, -0.18357513844966888, 0.03842579945921898, -0.30024591088294983, 0.6369478702545166, 0.051119089126586914, 0.2879526913166046, 1.7600326538085938, -0.05891022831201553, -0.6135963201522827, 1.385204553604126, -0.6627625226974487, -1.32509446144104, -0.28716644644737244, -0.10158748924732208, 2.0851404666900635, -0.9118812084197998, 0.249750018119812, 0.1764453500509262, -0.2786148190498352, -2.11505126953125, -0.3144233226776123, -0.22149433195590973, -0.8388954401016235, -1.7219284772872925, 0.8241492509841919, 1.5624198913574219, 0.011072593741118908, -2.3655588626861572, -0.4308617413043976, -1.2408599853515625, 3.3969881534576416, 0.4278171956539154, -1.630883812904358, 0.0018634966108947992, -1.543786644935608, -2.0496838092803955, 0.16480112075805664, -0.3783588409423828, 0.16695022583007812, 0.10472216457128525, -0.3260463774204254, -0.015005800873041153, -0.463259756565094, 0.5335226655006409, -0.16969868540763855, -4.461653232574463, -0.3002045154571533, 0.7196676731109619, 0.4342655539512634, 0.8259052634239197, -0.36834463477134705, 1.7521355152130127, 1.7788352966308594, 1.6102665662765503, 0.29692554473876953, 0.11164417117834091, 0.6400653719902039, -0.7923502326011658, -0.8561267852783203, -0.01883259229362011, -0.4126632809638977, -0.4722345769405365, -2.3313064575195312, 0.032056089490652084, -0.3357882797718048, 0.07440994679927826, 0.6275748014450073, -0.8811255097389221, -0.0002931449271272868, -0.004938064143061638, 0.7465866208076477, 0.4136349856853485, 0.6299980282783508, -0.4709542393684387, -0.6083419322967529, -0.133701890707016, -0.19967633485794067, -0.4289226830005646, 0.004890111740678549, -0.026712648570537567, 0.12076915800571442, -2.681873321533203, -2.452468156814575, -2.272096633911133, -0.011384418234229088, -0.5691709518432617, 0.2600604295730591, 3.281207323074341, -0.2816506028175354, -0.9098665118217468, 0.46854910254478455, -0.050984401255846024, -0.4043750464916229, 0.017265910282731056, -0.047164250165224075, -0.5723654627799988, -0.3673515021800995, -0.12623482942581177, -0.006963624153286219, -0.07402840256690979, -0.6445591449737549, 0.06098167225718498, 0.15675745904445648, -0.521342396736145, 0.398122102022171, -0.009758522734045982, 0.07076702266931534, -0.603424072265625, -0.6389926671981812, 0.10598728060722351, 1.4635515213012695, -0.46669724583625793, -0.43942001461982727, 1.2890247106552124, 0.23374895751476288, 0.1164475753903389, -0.5526103377342224, -0.6024854183197021, -1.7560880184173584, -1.8319263458251953, 0.6367623209953308, 0.7218189239501953, 0.03795100376009941, 0.029273027554154396, 0.9542883038520813, -0.9020358324050903, 0.050415024161338806, -0.3195125460624695, 0.10742994397878647, -0.056237637996673584, 0.001271724933758378, 0.24410884082317352, 0.36881783604621887, 1.4065775871276855, -2.4616572856903076, 0.2719617486000061, 0.0717177540063858, -0.8329710364341736, 0.11390653252601624, -0.5115712881088257, -0.010297980159521103, 0.5568630695343018, 1.2547266483306885, -1.2776075601577759, 0.30458131432533264, 0.7139760255813599, -2.0921761989593506, 0.44265347719192505, -0.2007019817829132, -0.5883722901344299, 0.45077866315841675, -0.7031157612800598, -0.4755941927433014, -0.36502212285995483, -0.8970946073532104, -0.023332951590418816, -0.7544912099838257, 1.932049036026001, -0.27184298634529114, 0.04365256801247597, -0.1442280262708664, 0.044137097895145416, -0.008601849898695946, 0.1894218921661377, -1.0970134735107422, -0.056963201612234116, -0.2620445191860199, 0.11683151125907898, 0.40868276357650757, -0.41656824946403503, -0.3863157033920288, -0.025518836453557014, -0.3091157376766205, -0.5902882814407349, 3.2058815956115723, 0.008481268770992756, 0.04127210006117821, -2.30765700340271, -0.020492209121584892, 0.6028220653533936, -1.8223891258239746, 0.7165512442588806, 0.31290513277053833, 0.08340226858854294, 0.9871899485588074, 1.1368457078933716, -0.24434661865234375, 0.6200753450393677, 0.5234302878379822, 0.6734972596168518, 2.1930596828460693, 0.2646234631538391, -0.0809875950217247, -0.18679171800613403, -0.09797024726867676, -0.09437893331050873, -0.5303874611854553, -0.6654960513114929, 0.3312675356864929, 0.21739992499351501, 1.1592779159545898, 0.49337589740753174, 0.6693323254585266, -0.6151403784751892, -0.08152120560407639, -0.45575016736984253, 1.6901462078094482, 0.1910981386899948, 0.14764787256717682, 1.0688211917877197, -1.0357197523117065, 1.0919519662857056, 1.1854091882705688, -0.0008096452802419662, 0.09713022410869598, 0.4682948589324951, -0.7486782670021057, -1.9089726209640503, -0.26409560441970825, -0.49901139736175537, 1.0132710933685303, 0.5714951753616333, 0.5580064058303833, -0.006075465120375156, -0.3889692723751068, 0.9238489270210266, -0.21526996791362762, -1.5927503108978271, -3.7194924354553223, -0.10439428687095642, 1.1264171600341797, 0.16871178150177002, -0.704030454158783, 0.06621279567480087, -1.5560791492462158, 1.2825442552566528, 0.9538331031799316, 0.22664789855480194, 0.30063799023628235, 0.35456234216690063, 0.13713987171649933, 1.1067088842391968, -0.8454252481460571, 0.8339052796363831, 0.05089297145605087, -2.162503242492676, -0.1759340614080429, -0.9498357176780701, -0.3652268648147583, 0.1874561458826065, -2.215743064880371, 0.5596046447753906, 0.36384689807891846, 0.06415276229381561, -0.0907730907201767, 0.38616207242012024, 0.12301401793956757, 0.29950907826423645, -0.15963047742843628, 2.287423610687256, 0.0035521399695426226, 0.3135184049606323, 0.5867899060249329, -0.017729153856635094, -0.29013901948928833, -0.29974374175071716, 0.006220982875674963, -0.3673839867115021, -0.010842201299965382, 0.14433196187019348, 0.30472779273986816, 2.30483078956604, 0.713710606098175, -0.14627663791179657, -0.5214591026306152, 0.5401268005371094, -0.5412874817848206, 0.1485716849565506, -0.24103017151355743, -1.924526333808899, 0.9494518041610718, -0.06692198663949966, -0.022715197876095772, -0.3905230760574341, 0.0704454854130745, -1.7280362844467163, -1.4567521810531616, 0.02625599317252636, -0.5752183198928833, -0.02985723502933979, 0.230544313788414, 1.4752660989761353, 0.009241722524166107, -0.22160355746746063, -0.521565318107605, -0.19631747901439667, 0.28170308470726013, -0.7279695868492126, 0.6684097051620483, 1.2680755853652954, -0.36423802375793457, 0.035458289086818695, 0.4264867901802063, -0.3251871168613434, -5.703013896942139, -0.05150734633207321, 0.6091482639312744, -0.06524419784545898, -0.7607507705688477, 0.15708084404468536, 0.06183551624417305, 0.11608042567968369, 0.6274011135101318, -2.6473145484924316, -1.5408930778503418, 1.20600163936615, 0.4243764877319336, -0.5078508853912354, -0.4067080020904541, 0.051477037370204926, 0.6241068840026855, 0.3233465552330017, -0.5646090507507324, 0.09006578475236893, -0.5400617718696594, 0.8160601258277893, -0.23029011487960815, 0.056317657232284546, 0.3677378296852112, -0.4352341890335083, -1.287523627281189, 2.9072165489196777, 0.2254852056503296, -0.16420947015285492, 2.770198345184326, 0.9141926169395447, -0.11417333781719208, 0.021410943940281868, -0.18259908258914948, -0.8619765639305115, -0.16392618417739868, -0.9815543293952942, 0.0021699785720556974, 0.518269956111908, 0.36667680740356445, 0.37573158740997314, -0.7401669025421143, -2.108734607696533, 0.0463249646127224, -1.4712636470794678, -1.285637378692627, 1.225612759590149, -0.6187702417373657, 0.12832556664943695, -0.23292966187000275, -0.043603766709566116, -0.6833903789520264, 1.7651618719100952, -0.06695611029863358, 0.00586931686848402, 0.44161131978034973, -0.394490510225296, -0.6270167827606201, 0.30924075841903687, 0.17801962792873383, 0.5482692718505859, -0.3379196226596832, 0.3906543254852295, 0.08186827600002289, 0.8224568367004395, -0.6980760097503662, 0.09121784567832947, -0.04789998382329941, 0.7061799764633179, -0.4284251928329468, -0.3293455243110657, 0.24985240399837494, -0.8473178148269653, -0.09306372702121735, 0.10432030260562897, 0.19243043661117554, -1.1165872812271118, 0.0842314213514328, -0.45214709639549255, -2.3125522136688232, -0.06282298266887665, 4.6861772537231445, -0.9307435750961304, 0.5729174017906189, 0.49246713519096375, 0.08094948530197144, 0.3686513304710388, 0.5749101638793945, -0.06322847306728363, 0.5082489252090454, 1.38297438621521, -0.12779150903224945, -0.35102152824401855, 0.15709711611270905, -0.3334691822528839, 1.2710810899734497, 0.5555225014686584, -0.48200783133506775, 0.08358606696128845, 1.1731188297271729, 0.0006787602906115353, 0.26849162578582764, 1.4367642402648926, -1.2647576332092285, -0.20405545830726624, -0.39648565649986267, -0.11174619197845459, 0.0791526809334755, -0.4925159215927124, -0.2598651647567749, 0.3868260383605957, 1.0762505531311035, 0.1827659159898758, -0.08784332871437073, 0.007891994901001453, -0.14748281240463257, 0.29700592160224915, 0.043471094220876694, -0.1890626847743988, -0.5187076926231384, -1.5975146293640137, -0.9812797904014587, -0.30762001872062683, 2.4415292739868164, 0.47652873396873474, 0.18507736921310425, 0.31392237544059753, -0.04477662593126297, -1.1044729948043823, 0.09832291305065155, -1.1588430404663086, -0.9108115434646606, -0.1339707374572754, -0.7508090138435364, -1.8548632860183716, 0.17023612558841705, -1.2525967359542847, 1.326323390007019, 0.06455599516630173, -1.8790459632873535, -2.3534977436065674, -0.9003489017486572, 0.5018606781959534, -0.1393665075302124, -0.21201485395431519, 0.027019077911973, -0.004818646237254143, 0.15052539110183716, 0.6360188722610474, -1.1548444032669067, 0.08243726938962936, 1.6809026002883911, -1.8412820100784302, 3.3855133056640625, 0.08100691437721252, -0.06853826344013214, 0.3137536942958832, -4.807962894439697, -1.025731086730957, -0.06551302969455719, 0.08681081980466843, 0.30564236640930176, 0.7532497644424438, -0.6861696243286133, 0.04531073570251465, 0.031508371233940125, 0.09218014031648636, -0.18627971410751343, -0.4383412301540375, -0.047194138169288635, -0.3931431472301483, 1.4621520042419434, -2.6574645042419434, 0.32320892810821533, 0.2589508593082428, -0.35183829069137573, -0.4366142153739929, -0.7524510025978088, -0.4518204629421234, 0.012228447012603283, 0.16181564331054688, 0.21211084723472595, 1.4657427072525024, -0.453416645526886, 0.6462466716766357, -0.4710422456264496, 0.2351330816745758, -0.050252921879291534, 0.2163437455892563, 0.1525285840034485, 1.2051410675048828, 0.06676048785448074, 0.04540185257792473, -0.6941068768501282, 0.17051224410533905, -1.223983645439148, -0.10166527330875397, -1.1716288328170776, -0.018116358667612076, 0.8653193712234497, -0.480093389749527, 3.071361541748047, -0.23456615209579468, 0.1715317815542221, -0.028292229399085045, -0.7682609558105469, 0.21014678478240967, -0.574802815914154, -1.7642009258270264, -0.1937446892261505, -0.6191701292991638, -0.04407977685332298, 1.137393832206726, -1.1834052801132202, 0.3865848779678345, 0.02107354812324047, -0.09335598349571228, -0.0594124011695385, -1.3541209697723389, 0.15148138999938965, 0.5869385004043579, 1.8130897283554077, -0.2426784336566925, -0.4346661865711212, -0.09247834235429764, 0.25208529829978943, 1.217857003211975, 0.2305285781621933, -0.7721759080886841, 0.28987833857536316, -0.43544748425483704, -0.61434006690979, 0.1314820945262909, 0.517749547958374, 0.06267333775758743, 0.7468968629837036, 0.8379361033439636, 0.5209673643112183, 0.23522204160690308, 1.6578880548477173, 0.738980770111084, 0.7933533787727356, -0.9484727382659912, -0.21978497505187988, -0.5242013335227966, 1.8173434734344482, 0.07162406295537949, -0.3884277045726776, 0.4751020669937134, 0.13281658291816711, 0.04432762786746025, 0.08706238865852356, 1.1602712869644165, -0.9764683246612549, -0.47286444902420044, 0.5075910091400146, -0.10262753814458847, -0.5154819488525391, 0.31864693760871887, -1.9155279397964478, -0.019388902932405472, 1.904693841934204, -1.1588823795318604, -0.26749077439308167, -0.005157234612852335, -0.1750425100326538, 2.087813377380371, 0.036547619849443436, 0.3637661635875702, -0.15208935737609863, 1.6238363981246948, -1.3536732196807861, 2.974278688430786, 0.34065553545951843, -0.015211853198707104, -0.3089360296726227, -0.14454880356788635, 1.5610417127609253, 0.5217775106430054, -0.23895063996315002, -0.6392037272453308, 0.38645970821380615, 0.6892803311347961, 0.5224267840385437, -2.0764636993408203, -0.1661016047000885, 0.3469797968864441, 2.14060115814209, 0.3227367103099823, -0.6389076113700867, -0.06098838895559311, -0.0008806159603409469, -0.003036721609532833, -0.006563662551343441, -2.2843685150146484, -0.4423728585243225, 0.09968946874141693, 0.3166241943836212, -0.6237770318984985, -1.0845476388931274, 1.4673492908477783, 0.0026793386787176132, 0.29753366112709045, 1.7902475595474243, -1.6966960430145264, -0.19651615619659424, -0.8702462315559387, -0.1400156319141388, 0.033986590802669525, -0.39660781621932983, -0.2604871988296509, -0.07725363969802856, -0.05672626942396164, -0.768212080001831, -0.28624188899993896, 0.1913316696882248, 0.3135458827018738, 0.058783117681741714, 0.15098582208156586, 0.03008881025016308, 0.16946181654930115, -0.33865419030189514, 1.623457670211792, -0.6694610118865967, 0.12911461293697357, 1.6670817136764526, -0.025182602927088737, 0.059840112924575806, -1.221393346786499, -0.18804147839546204, -0.8881572484970093, 0.6218773126602173, 0.008205394260585308, 0.20477239787578583, -0.07076304405927658, -0.3510625660419464, 0.22373272478580475, -0.08345123380422592, -0.16673080623149872, -0.021874820813536644, 0.22029902040958405, -0.5836589336395264, -0.004959242418408394, 1.1477006673812866, 1.600590705871582, -0.18252894282341003, -0.17951767146587372, -0.5145230889320374, -1.6614781618118286, -0.6412814855575562, -0.5707666277885437, -1.0360161066055298, -0.7764044404029846, -0.25153738260269165, 0.026770131662487984, -0.34979134798049927, -0.35620924830436707, -0.847511887550354, 0.049188677221536636, -0.9812293648719788, -0.038775451481342316, -1.646480917930603, -0.23260550200939178, 0.017065342515707016, -0.41532835364341736, 0.511550784111023, -0.07059495896100998, 0.2696658968925476, -0.01100071705877781, 2.197807788848877, 1.897130012512207, 0.005307320971041918, 0.5610105395317078, 1.7810624837875366, -0.10175095498561859, 0.19888192415237427, -0.01655990071594715, 0.9676375985145569, -0.5713534355163574, 0.053173597902059555, 0.7730891704559326, -0.0004961875383742154, -2.462958574295044, 0.22306136786937714, 0.0628422275185585, 0.7998009920120239, 0.4730296730995178, -0.33858489990234375, 1.4940439462661743, -1.231441855430603, -0.09041564166545868, 0.3469141721725464, 0.06749187409877777, -2.0297281742095947, -0.3343037962913513, 0.11210687458515167, -1.0839356184005737, -0.009502752684056759, -0.32361048460006714, 0.023251084610819817, -1.306012749671936, -0.2818164825439453, -1.1024501323699951, -0.7026425004005432, -0.1523543745279312, -0.003288197098299861, -0.9331183433532715, -0.44581303000450134, 0.09071909636259079, -0.2821621894836426, -2.2489333152770996, 1.3196454048156738, 0.18158429861068726, -0.3224882483482361, 0.04883132502436638, -0.14355705678462982, 0.6980224251747131, 0.02058444172143936, -1.1832661628723145, 0.006409102119505405, 1.0156886577606201, -0.4655599594116211, 0.889628529548645, -0.08277343958616257, 1.55209481716156, 1.8949607610702515, -0.9991448521614075, 0.020169001072645187, -0.08105972409248352, -0.7278522253036499, -1.1770179271697998, -0.3390266001224518, 1.2627781629562378, 0.5396542549133301, -0.8919222354888916, -0.7841883897781372, -0.47812771797180176, -2.3769278526306152, -0.006384565494954586, 0.5043896436691284, -1.7945756912231445, -0.11261985450983047, 0.2588004171848297, 0.07633606344461441, -0.21775802969932556, 0.48304232954978943, 0.10858616232872009, 0.03145802766084671, -0.021588411182165146, 0.043384164571762085, -0.517316460609436, -0.2722252905368805, -0.029337909072637558, 0.3491247892379761, -0.0042753336019814014, -2.0546088218688965, -0.4805278480052948, 1.402909755706787, -0.0649184137582779, 0.22868843376636505, -0.07243726402521133, -0.18263153731822968, 0.23675502836704254, 3.1541593074798584, -0.26446956396102905, 1.0222594738006592, 0.1421775370836258, 1.8266892433166504, 0.007668493781238794, 0.5638758540153503, -0.13636749982833862, -0.10483213514089584, -1.2736151218414307, 1.0014539957046509, -1.1133123636245728, -0.5902751684188843, 1.5004240274429321, 1.3945949077606201, 0.9597298502922058, 2.521717071533203, -0.8368806838989258, -4.050974369049072, 0.7939534187316895, -0.048587117344141006, 0.9039487838745117, -0.2926679253578186, -0.5358238220214844, -0.5258659720420837, -0.0216544009745121, 1.4299331903457642, -0.00306988088414073, -0.08796320110559464, -0.017399467527866364, -0.12414347380399704, 0.023316478356719017, 0.9615311622619629, -0.9860577583312988, -0.5591352581977844, -0.39204853773117065, -0.16171593964099884, -1.406999945640564, 0.4423193335533142, -0.09222432971000671, -0.033533576875925064, -1.6533738374710083, -0.15437553822994232, -0.7598598599433899, -1.229580283164978, 0.6134130358695984, -3.811021566390991, 0.5334135293960571, 0.1402560919523239, 0.18733671307563782, 1.1762149333953857, -0.06686176359653473, 3.443934679031372, 0.1342606097459793, -0.01974644884467125, 0.508843183517456, 0.6939855813980103, 0.185268372297287, 0.009420603513717651, -0.13539279997348785, 0.22139835357666016, -1.2057857513427734, 0.19720706343650818, 1.0167499780654907, 0.605055034160614, -0.11007191985845566, -0.027265410870313644, 1.2384406328201294, 2.004836082458496, 3.1873624324798584, -0.11192485690116882, -0.04312695562839508, -1.2311022281646729, 0.1965818703174591, -2.352726936340332, 0.01836561970412731, 0.353147953748703, -1.0469763278961182, 1.221685767173767, -1.4706027507781982, 0.0362657830119133, 0.6577743887901306, 0.5300700068473816, 0.10288282483816147, 0.05952185019850731, 0.06620731949806213, -0.9694485068321228, -0.04738445580005646, -1.114643931388855, -0.5604920983314514, 0.8759246468544006, -0.2956348955631256, -0.017158564180135727, -0.3072372376918793, 0.12220796197652817, -1.2494874000549316, 0.10306499153375626, 0.09880229830741882, 2.085094451904297, 0.08717399090528488, 0.5519351959228516, -0.18047751486301422, -0.44990143179893494, 0.033661797642707825, -0.07897558808326721, 0.01052077766507864, -0.8431814908981323, -0.38383355736732483, -0.030175236985087395, 0.6809306144714355, 1.548614740371704, 0.6666209101676941, -0.4970674514770508, 0.8775227665901184, -0.07719655334949493, -2.424240827560425, 1.4909710884094238, 0.058072224259376526, -0.30078402161598206, 0.10813727229833603, 0.0014073083875700831, 0.0014076493680477142, -0.9083831906318665, 0.1597888320684433, 1.389140248298645, 0.32313624024391174, -0.27760550379753113, -0.027043495327234268, -0.015523724257946014, 0.04001568257808685, 1.0723849534988403, -0.0004943539388477802, 3.5708446502685547, -0.0013956156326457858, -0.2892751693725586, 0.18750828504562378, 0.009056057780981064, -0.0015793552156537771, 0.099496990442276, -0.5683010816574097, 1.2045633792877197, 0.03035891242325306, -1.027672290802002, 0.32397252321243286, -0.5502718091011047, 0.23430155217647552, 0.9284105896949768, -0.0463964119553566, 0.8320842385292053, -0.25736650824546814, -0.004412907641381025, 0.3804425597190857, 0.9366034865379333, 0.9640527367591858, 0.07953379303216934, 0.6700101494789124, -0.06895005702972412, -1.3406915664672852, 0.05165713280439377, 0.24238692224025726, -0.1941869705915451, -0.35869574546813965, 0.06903862208127975, 0.7213438749313354, -0.2196107655763626, -0.7830146551132202, -1.104753851890564, 0.5111892223358154, -0.5526495575904846, 0.9530912041664124, -2.5229082107543945, -1.820971965789795, 0.09014926850795746, -0.21001064777374268, -0.09376538544893265, -2.063727617263794, 0.31093019247055054, 0.7568334341049194, -1.6554864645004272, 0.08770155906677246, -0.01703699678182602, -0.2972831726074219, -0.2721240818500519, -3.7115819454193115, 0.23223777115345, 0.27365079522132874, 0.5701831579208374, -0.5851037502288818, -0.6072897911071777, 0.1761263906955719, -1.5930083990097046, -0.21948261559009552, 0.06991074979305267, 0.4913315773010254, 0.058094460517168045, -0.969335675239563, 1.1732211112976074, -0.32248765230178833, -0.5491048693656921, 0.4771607220172882, 0.7703424692153931, 0.48182621598243713, 0.7308769822120667, -0.04245278239250183, 0.14472442865371704, -0.030628347769379616, 0.0939907655119896, 1.312517762184143, -0.10074091702699661, -0.45766088366508484, -0.775424063205719, -1.3303097486495972, -0.15377545356750488, 0.32182157039642334, 1.8675769567489624, -0.5404695272445679, 0.3000890910625458, 0.047703925520181656, -1.5605107545852661, 0.12726691365242004, 0.40543320775032043, -1.0371544361114502, -0.39699146151542664, -0.3941584527492523, -0.009453210048377514, 1.330639123916626, 0.11351016163825989, 0.023979604244232178, -0.07061364501714706, 0.5605612993240356, 0.09201771765947342, 0.6363327503204346, 0.4932136833667755, -0.689384937286377, 0.05533698946237564, -2.842205286026001, -1.4050252437591553, 1.0140819549560547, -0.48632845282554626, -0.19802193343639374, -0.5934368371963501, -0.345761239528656, -0.010440326295793056, -0.5104657411575317, -0.9176127314567566, 0.3059675097465515, -0.07433836907148361, 0.20361217856407166, 2.5998740196228027, 0.18787039816379547, -0.12292438745498657, -2.1041040420532227, 1.1011035442352295, 0.024581560865044594, 0.5142163634300232, -1.6935110092163086, -0.3274487257003784, 1.598405122756958, 1.8941196203231812, 0.2339513599872589, 0.3801530599594116, 0.9065349698066711, 0.7577649354934692, -0.12276763468980789, 0.13315819203853607, 0.8902451992034912, -1.7286409139633179, 0.33378034830093384, 0.2949191629886627, 0.8280547261238098, -0.8549503684043884, 0.012566463090479374, -0.338646799325943, 0.7166769504547119, 1.5636203289031982, -0.3119610846042633, 0.17828388512134552, -0.2655021548271179, 0.7523181438446045, 0.2828257381916046, -0.30111992359161377, -2.591567277908325, 2.5467193126678467, 0.038097772747278214, 0.08112357556819916, -0.13599053025245667, -0.15279237926006317, 1.1241012811660767, 0.0961005762219429, 0.7067803740501404, -1.5201952457427979, -0.044795744121074677, 0.3454723656177521, 0.004004951566457748, 0.298951119184494, 0.32171449065208435, -0.15459643304347992, 1.6429696083068848, -0.4174489378929138, 0.24298562109470367, -0.23596271872520447, 0.4022141993045807, -0.07036540657281876, 0.43995434045791626, 0.1448548287153244, -0.0683300718665123, -0.07001065462827682, -1.1936429738998413, 0.18817247450351715, 0.9277594089508057, -0.5419226884841919, -0.7381185293197632, 0.08363599330186844, 1.0691324472427368, -0.6007399559020996, -1.099274754524231, 0.429413378238678, 0.43383529782295227, 1.3438646793365479, -0.3332667052745819, 0.8324768543243408, -0.157080739736557, -0.15864118933677673, 0.3041396141052246, -1.489518404006958, 0.5329040288925171, 0.05632789805531502, 0.11792667955160141, 0.15855814516544342, -0.8182078003883362, -0.014435010962188244, -0.6691758036613464, -1.4640575647354126, -2.1382527351379395, 0.3512844145298004, -1.2072185277938843, -0.4585753083229065, 0.11487168818712234, 0.1397145688533783, -0.5475624799728394, 0.0816425085067749, 1.0158357620239258, 0.06114525720477104, -0.14922423660755157, 0.44460201263427734, -0.030080407857894897, 0.040789492428302765, -0.03823259472846985, -0.8321142196655273, -0.02861712872982025, 0.018047941848635674, 1.724184513092041, 1.0093693733215332, -1.4494385719299316, -0.6862903833389282, -0.10288215428590775, 0.4875852167606354, 0.20173729956150055, -1.1367597579956055, 1.2619882822036743, -0.24455656111240387, 0.6240009069442749, 1.5350357294082642, -0.6776801347732544, -0.049708738923072815, -0.6704392433166504, -0.1489475518465042, -0.21335820853710175, -3.9033985137939453, 0.4672686457633972, 1.6819872856140137, 0.05512571334838867, -0.023731090128421783, 0.5536181926727295, 0.23336274921894073, -2.6098082065582275, 0.48844751715660095, 0.7586559057235718, 1.9375783205032349, 0.8196204304695129, 0.016378311440348625, -0.055128373205661774, 0.6312296986579895, 0.7130212187767029, 0.5827294588088989, -0.46823886036872864, -0.06490561366081238, 0.09226350486278534, 0.15473152697086334, 0.8616729378700256, 0.15117092430591583, 1.6473302841186523, 0.02802550047636032, -0.7926207184791565, 1.1561763286590576, 2.354541540145874, -0.31337836384773254, -0.7757940292358398, -0.030392134562134743, -1.4656027555465698, 0.029962506145238876, 0.6236645579338074, 2.7622509002685547, -0.011865736916661263, 0.25286057591438293, -0.05052953585982323, -0.6512183547019958, -0.11731315404176712, -2.1155664920806885, -0.859293520450592, 0.18494747579097748, -2.3088982105255127, 0.6793519258499146, -0.5725831985473633, -1.5050181150436401, -0.00012473844981286675, 1.3593412637710571, 0.10933324694633484, 0.823967456817627, 0.47131282091140747, 1.1920280456542969, 0.4287630021572113, 0.006313336081802845, -0.9570383429527283, 1.5592594146728516, 1.3280924558639526, 0.5891993641853333, -0.32790881395339966, 0.000158278489834629, -5.819617748260498, 0.5420604348182678, 2.276620388031006, -0.3868728578090668, -0.013767186552286148, 0.49624186754226685, -0.5540112853050232, -0.794494092464447, 0.21789871156215668, 0.17705510556697845, -0.632522463798523, -0.9141022562980652, -0.3914715647697449, 0.8195573091506958, 0.15099529922008514, -0.024081680923700333, 2.086973190307617, -0.010682825930416584, 2.335052728652954, 0.8024972081184387, 1.1312617063522339, 0.02942875772714615, -0.05663551017642021, 0.6831536293029785, 0.025087492540478706, 1.0645790100097656, 0.023415783420205116, -0.6108907461166382, 0.06189354509115219, 0.664533793926239, -0.0997171700000763, 1.844162940979004, -0.06942618638277054, -0.11538373678922653, -0.12134073674678802, 1.7906029224395752, -0.960282564163208, 0.2628299295902252, 0.004371149931102991, 1.8955284357070923, 0.08436353504657745, -0.5274657607078552, -0.30035319924354553, -0.006537486799061298, -0.14308203756809235, 0.015579844824969769, -0.4198618531227112, 2.570727825164795, 1.083726406097412, -0.09590717405080795, 0.2705098092556, 0.06820088624954224, 0.3147106468677521, 2.32316517829895, 0.08505529165267944, 0.22961091995239258, 0.41412338614463806, 2.309180498123169, 0.012976405210793018, 0.20255550742149353, -2.713505983352661, -0.007045579608529806, -0.011422215029597282, -0.19604308903217316, -0.4157300293445587, -0.40788349509239197, -0.008670622482895851, -2.2164056301116943, 0.532852053642273, -0.9485365748405457, 0.9225417971611023, -0.4742113947868347, -0.4686281085014343, -1.0586774349212646, 0.016131581738591194, -0.07510823756456375, 0.4963492751121521, -0.0007996116182766855, -0.2141987681388855, -0.4079732596874237, -1.16065514087677, 1.3669019937515259, -0.4645823538303375, -0.15221300721168518, -1.0510461330413818, -0.12356234341859818, 0.1020534336566925, 0.044597819447517395, -0.4430585503578186, 0.964571475982666, -0.020676763728260994, -0.1353510618209839, 2.7559616565704346, -0.5889046788215637, 1.035456895828247, -0.001580153126269579, -1.815989375114441, 0.19927026331424713, 0.03474538028240204, -0.06125956401228905, 0.2746797800064087, -2.5387046337127686, -0.21228158473968506, -0.7076495885848999, -0.4219377338886261, -0.042535264045000076, -0.3225816786289215, -2.4936065673828125, 1.8947982788085938, -0.5029539465904236, -0.4434405267238617, 0.8752257823944092, -2.5421009063720703, -1.405116319656372, 0.21862047910690308, 0.9147531986236572, 0.21295438706874847, 0.35582515597343445, 0.602756917476654, 0.5374372601509094, -0.19529934227466583, 0.9163240790367126, 0.09708043932914734, 0.06933572888374329, -0.8232277631759644, -0.5525564551353455, -0.5373634696006775, 0.041371628642082214, 0.09900999069213867, 0.6625820994377136, -0.9003424048423767, 0.42707890272140503, 0.317995548248291, -0.022030001506209373, -0.7398102283477783, -1.1266037225723267, -0.6333824396133423, -0.6409165263175964, 0.20497721433639526, 0.03412175178527832, 0.40445706248283386, 0.448682576417923, 0.02051757089793682, 0.5544461607933044, 2.062685489654541, 0.3518097698688507, 1.9014465808868408, 0.3030841648578644, 0.9203846454620361, 1.7190428972244263, -0.02328602597117424, 0.029929298907518387, 0.0060468376614153385, 1.9720200300216675, -0.013027294538915157, 0.8029026985168457, -0.1450449675321579, -0.366415411233902, 1.3933701515197754, -0.3678077757358551, 0.43737486004829407, -1.3762462139129639, -0.3050965666770935, -0.7695793509483337, -0.42791947722435, 0.030716797336935997, -0.04851322993636131, 0.11207333952188492, 0.18036198616027832, 0.4037591814994812, 0.09587807953357697, -0.07289435714483261, 0.8291701674461365, 1.0748037099838257, -0.5900485515594482, -0.6969420909881592, -1.1403559446334839, -0.0226404070854187, 0.5076489448547363, -0.015597354620695114, -0.11907558888196945, -0.6956964135169983, -0.8282470107078552, 0.10513048619031906, -2.1955745220184326, 2.0226244926452637, 0.023939212784171104, -0.4996374845504761, 0.45747193694114685, -0.1990000158548355, 2.244337320327759, -0.12245213985443115, 0.24707378447055817, 0.20953236520290375, -0.19612358510494232, 1.224472999572754, -0.016333062201738358, 0.5878457427024841, -0.3104584813117981, -0.15539878606796265, 1.0396493673324585, -0.08912084996700287, 2.6096625328063965, -0.4685831367969513, -0.2117503136396408, 7.495883464813232, 0.026385772973299026, -1.0993753671646118, 0.03710893541574478, 0.6721976399421692, 0.8403090834617615, -0.881504237651825, 0.24932457506656647, 0.033684734255075455, -0.010480904020369053, -0.9590522050857544, 1.2242426872253418, -2.1144843101501465, 0.4405927360057831, 0.43061965703964233, -0.42052799463272095, 0.5093386173248291, 0.3647809624671936, 0.5778214931488037, 1.1623371839523315, -0.21007013320922852, -0.714137077331543, -0.034287601709365845, 1.0430220365524292, 0.41675207018852234, -0.02768273837864399, -0.18714427947998047, 0.16319948434829712, -0.6325781941413879, -0.13608506321907043, 0.023347845301032066, -0.7010753154754639, 0.3774931728839874, -2.893448829650879, -0.31657353043556213, 0.17578835785388947, -0.9284092783927917, -0.665034294128418, -0.38104408979415894, 0.04644182696938515, -0.8843785524368286, -0.3949228525161743, 1.7247470617294312, -0.11859171092510223, -1.2050453424453735, 0.163756862282753, -0.04697967320680618, 0.0354621596634388, 0.11099770665168762, 0.5796663165092468, -0.22343744337558746, 0.25342339277267456, 0.24588993191719055, -0.5229768753051758, -3.8009705543518066, 0.004003584384918213, -0.019895058125257492, -0.01616823300719261, 0.30256715416908264, -0.0852922573685646, 1.579895257949829, 0.015573490411043167, 0.2341403216123581, -0.19960977137088776, 0.3382717967033386, 0.13828420639038086, -2.413825511932373, -0.9960218071937561, 0.8855838179588318, -2.34598445892334, 0.3980550467967987, -0.39048832654953003, -0.4462206959724426, 0.1378818154335022, 0.03653378784656525, -0.04329908266663551, 0.32788506150245667, 0.3162005841732025, 2.031681537628174, 0.364938884973526, -0.8791666626930237, 0.45322543382644653, -0.5589569211006165, 0.11748890578746796, -2.471005916595459, -0.7272522449493408, 0.023046264424920082, 0.11835097521543503, -0.9442493915557861, -0.19968624413013458, 1.346839189529419, 0.11069737374782562, 0.05080682411789894, 0.7859184741973877, -0.2273431420326233, -0.19957515597343445, -1.142717719078064, -0.9099370837211609, 0.07821787893772125, 1.5730986595153809, -0.9175719618797302, -0.3455008864402771, 0.0250487569719553, 0.24073177576065063, -0.37825921177864075, 0.3900708258152008, 0.1122586727142334, 0.7072705626487732, -0.05115983635187149, 0.36127129197120667, 0.28350797295570374, -0.10769598931074142, -0.9671096205711365, -1.022414207458496, 0.36374178528785706, -0.9877684116363525, 0.8506515026092529, 0.31907764077186584, -0.26319342851638794, -2.000270366668701, -0.8553870320320129, 0.11970265209674835, 2.6889188289642334, 0.05456953123211861, -0.25112661719322205, -0.025180652737617493, -0.26318618655204773, -1.2232521772384644, -1.3477073907852173, 1.1224831342697144, 0.2393747717142105, -0.03552505746483803, 0.5142391324043274, -0.49913290143013, -0.060263145714998245, 0.6532227396965027, 0.2521640658378601, -0.21919836103916168, -0.33364060521125793, -0.420987069606781, 0.268015593290329, 0.35722726583480835, 0.10011866688728333, -1.1804476976394653, -0.9839659333229065, -0.11699538677930832, -1.0767749547958374, -0.20396652817726135, 0.36122605204582214, -0.4870106279850006, 0.9411888718605042, -0.6310693621635437, -0.34750503301620483, -0.13700155913829803, 1.1700645685195923, -0.4609578847885132, 0.03237991780042648, -0.06539043039083481, 0.2518330216407776, 3.726278781890869, 1.1919310092926025, 0.010083165019750595, -0.24004873633384705, -0.8431081175804138, 0.7563261389732361, 1.2740744352340698, -0.12047334015369415, 0.5894985198974609, -0.8010677099227905, -0.34046679735183716, 0.38373810052871704, 0.03880993276834488, -0.06493176519870758, 0.9595381021499634, -0.05914066359400749, -0.8185047507286072, 0.11505818367004395, -0.3908418118953705, 0.8739768266677856, 1.4987809658050537, -0.7712733745574951, -0.054123297333717346, -0.005022591445595026, 1.0205459594726562, 2.361985921859741, -0.15775112807750702, 0.011339206248521805, -0.5941827893257141, -0.014767062850296497, 0.21897751092910767, 0.054965414106845856, -0.004197011236101389, 0.08796710520982742, 0.6123957633972168, -0.0690310150384903, 0.13693641126155853, -0.28112223744392395, -0.20556402206420898, -0.0021610299590975046, -1.050563097000122, 0.6004418730735779, -0.2304813116788864, 0.044077374041080475, 0.358230859041214, 1.1536766290664673, 0.5515872836112976, 0.3731129765510559, -1.5078800916671753, -3.200460910797119, 0.08016468584537506, 0.8027581572532654, 0.6766387224197388, -1.12722647190094, 0.0393371544778347, 0.0729389414191246, 0.0022719567641615868, -0.09861023724079132, -1.8959084749221802, -0.6369857788085938, -0.7205816507339478, -0.04302985593676567, -0.0612921342253685, -0.6971065402030945, -0.29163658618927, 0.122042216360569, -0.5842276811599731, 1.139448642730713, 1.3149363994598389, 0.07191157341003418, 0.40898633003234863, 0.1543952375650406, -0.45113605260849, 0.4834083318710327, -0.43284156918525696, -0.011396657675504684, 0.18150898814201355, 0.12243952602148056, 0.7603721022605896, -0.5911301374435425, -0.2364731729030609, -0.1640935242176056, 1.684774398803711, -2.271995782852173, 0.8354821801185608, 3.5721538066864014, 1.2366896867752075, -0.13144709169864655, -0.13436788320541382, 0.21201938390731812, -0.9663893580436707, 0.8695226907730103, -0.8032826781272888, -1.0318466424942017, 0.32400691509246826, -0.3621799945831299, -1.1961034536361694, -1.9773685932159424, 0.05458848178386688, 0.17459696531295776, -0.0872335210442543, -0.5576767921447754, -1.1030490398406982, -0.8157129883766174, 0.9158594608306885, -0.023249706253409386, -0.27892735600471497, -0.9325635433197021, 2.610375165939331, -0.20801717042922974, -3.778174877166748, -0.2283661663532257, 0.23801057040691376, 1.1381109952926636, 0.05787105858325958, 0.39687126874923706, -1.715414047241211, 0.07211040705442429, 0.24534299969673157, -1.8953007459640503, -0.0050243656150996685, -0.15327364206314087, 1.1201796531677246, -0.058385562151670456, 0.3182290196418762, 0.22336238622665405, -0.11604390293359756, -0.7810862064361572, 0.002117748837918043, 0.15270459651947021, 1.2615103721618652, -0.11942987889051437, 0.9036477208137512, 0.00571313314139843, 0.3806663751602173, 3.682898998260498, 0.5083901882171631, -0.0021635424345731735, -0.14619942009449005, 0.0023454236797988415, -0.07831140607595444, -0.06675968319177628, 0.8612315058708191, -1.7227076292037964, -0.17835883796215057, 0.5592208504676819, 0.3129813075065613, -0.050511498004198074, -0.8734796643257141, 0.12717948853969574, 0.651714026927948, -1.310716986656189, 0.06178274378180504, 0.19983479380607605, -0.17241090536117554, -0.039765726774930954, -0.5174341201782227, -0.2040465623140335, -0.017447076737880707, -0.5938519835472107, 0.008803136646747589, -0.14376451075077057, -0.03831764683127403, 0.5167107582092285, -0.015446430072188377, 0.23325762152671814, -0.8488356471061707, 0.05131836608052254, 0.6944264769554138, 0.6574661731719971, 1.9729136228561401, -0.3123854696750641, -0.5379797220230103, -3.579432487487793, 0.07145470380783081, -0.3698028028011322, 0.013740192167460918, 1.8949997425079346, 0.39977943897247314, -0.3146864175796509, -0.855053186416626, -0.05109370872378349, -0.2894483506679535, -0.1076325923204422, -1.0543677806854248, -0.31823718547821045, 0.24183516204357147, -0.23971842229366302, 0.11185581982135773, -0.33977586030960083, -0.5575977563858032, -3.738579750061035, -0.2724757790565491, 0.5768658518791199, 0.1977500170469284, -0.5637909173965454, 0.7568414211273193, -0.6080570816993713, 0.7706754803657532, -0.6431649923324585, -0.13475337624549866, 0.2564353346824646, 0.6332383155822754, 2.173199415206909, 0.8215148448944092, 0.28091514110565186, -0.7778763175010681, -0.12905126810073853, 0.7543777823448181, 0.30191516876220703, 0.9184499382972717, -0.9862061738967896, 0.14163289964199066, 0.8665846586227417, -0.5712453126907349, -0.18511776626110077, 0.03430896997451782, 0.13334350287914276, 0.6314921975135803, -0.2999011278152466, 0.23618079721927643, 0.01361184660345316, 0.13444048166275024, -0.034614212810993195, -2.070493459701538, -0.5319553017616272, 0.22039444744586945, -0.26330363750457764, -0.21006640791893005, -0.09521303325891495, -0.6103209257125854, -0.010641786269843578, 0.00665822671726346, -0.39148449897766113, 2.5044362545013428, -0.678052544593811, -0.4240573048591614, -0.2678115665912628, -0.27698954939842224, 0.07149121910333633, -0.07731617987155914, 0.01582411117851734, -0.4148051142692566, 2.2685930728912354, -0.20754598081111908, 0.350393682718277, -0.20978949964046478, 0.5187515616416931, -0.108118437230587, -0.7735815644264221, -0.36352238059043884, -0.43390780687332153, -0.7852376699447632, -0.0010245259618386626, 0.09165801852941513, 0.02630672976374626, 0.21568244695663452, 0.009818732738494873, -0.28148987889289856, 0.15881691873073578, 2.216139554977417, -0.5459498763084412, -0.35253268480300903, 0.18996836245059967, -0.03938343748450279, -0.07442395389080048, 1.010468602180481, -0.5671446919441223, -0.18168304860591888, -0.06854813545942307, -1.4202003479003906, -0.022808102890849113, -0.0988953486084938, -1.9465522766113281, 1.1506600379943848, -0.38808292150497437, 0.12112586945295334, 0.15685902535915375, -2.3605880737304688, 0.5353749394416809, 4.3768229484558105, 0.0033518436830490828, 1.7747424840927124, 1.09954833984375, 1.2258909940719604, -0.5675247311592102, 0.016889648512005806, -0.8345590829849243, 0.1689188927412033, -0.027153316885232925, 1.3989464044570923, 1.0438389778137207, -2.021306276321411, 0.7342729568481445, 1.8372538089752197, 0.8179086446762085, -0.19741995632648468, -0.06184840947389603, 0.6228208541870117, -0.10085903108119965, -0.8735053539276123, 1.1720614433288574, -0.4871314465999603, 0.35503089427948, -1.685579776763916, -0.32460135221481323, -0.10939457267522812, -0.4326328635215759, 0.4882272779941559, 0.6461262106895447, -0.14562907814979553, 0.14278490841388702, 0.21878540515899658, 0.01426070649176836, 0.30340468883514404, 1.10381281375885, 1.1177775859832764, -0.5883328914642334, -0.09818582236766815, 0.12921947240829468, 1.9440261125564575, 1.2734169960021973, -0.44641780853271484, 0.1190129891037941, -0.6247401237487793, 0.28386008739471436, -0.20574870705604553, -0.04132096841931343, 0.6002708077430725, -0.4803178310394287, -0.11409620940685272, -0.41269224882125854, -2.4883503913879395, -0.02503972314298153, -1.3343855142593384, 0.14334577322006226, 0.5046787858009338, 0.5304615497589111, -1.212485909461975, 0.6802688241004944, -0.379218727350235, -0.8594323992729187, 0.5700464248657227, -1.920928716659546, -0.1919226348400116, 0.7110620141029358, -0.14224128425121307, 0.5022221803665161, -0.19466429948806763, -1.2729383707046509, 0.5128101110458374, 0.0760713741183281, -1.2375071048736572, 0.1114191859960556, -0.8749206066131592, 0.0921933725476265, -3.29341459274292, -2.410963535308838, 0.09839906543493271, 0.0426773764193058, 1.675551414489746, -1.609119176864624, 0.14683856070041656, 0.14015744626522064, -0.8260124921798706, -0.027423035353422165, -0.03660854697227478, -0.04761432111263275, 0.009619394317269325, 1.0791879892349243, -0.7515954971313477, -0.2497795820236206, -0.11699409037828445, -0.20152127742767334, 0.012460997328162193, 2.7702736854553223, 0.11393105238676071, 0.1767188161611557, 0.7535994648933411, -0.4366609454154968, -1.041414499282837, 0.035718128085136414, 0.08716617524623871, -1.1501246690750122, 0.889419436454773, 0.0005274746217764914, -0.05855390056967735, 0.3885384202003479, 0.12079919129610062, 0.7854908108711243, 2.8096020221710205, 1.0753173828125, 0.021253420040011406, 0.17317715287208557, 0.12377436459064484, -0.22280675172805786, 1.0023877620697021, 0.10016254335641861, 0.34231024980545044, -0.5797218084335327, 0.6552527546882629, 0.4051969349384308, 2.0919852256774902, 1.7478828430175781, 0.1402190923690796, 0.14093951880931854, 0.0581069216132164, -0.6957343220710754, 0.059413060545921326, 0.7218372225761414, 0.9128491282463074, -0.08587577193975449, -0.14839981496334076, 0.014254682697355747, 0.04795593395829201, -0.3641033470630646, -0.3715871572494507, 0.1255035400390625, 0.4961940348148346, 0.6981843709945679, -0.81695157289505, -0.5292447805404663, 0.7977599501609802, 2.4199461936950684, -2.768109083175659, 0.9062514305114746, 0.07076667249202728, -0.8472974300384521, 0.29756447672843933, 0.26969969272613525, -0.2626282572746277, 0.06402815133333206, -0.7795599102973938, 1.2992558479309082, -0.05400931090116501, 0.47668197751045227, -0.5232099890708923, 0.4916733503341675, 0.017843512818217278, -0.020148899406194687, -1.156987190246582, -0.7429687976837158, 0.7503412961959839, 0.636826753616333, 0.6780864000320435, 1.7381644248962402, -0.02069670706987381, -1.3197376728057861, 1.6258618831634521, -0.6696799397468567, -2.199044942855835, 0.05740591883659363, -0.5249617695808411, -0.10351186245679855, -0.33483290672302246, 2.6314661502838135, 0.005744241178035736, 0.6215621829032898, 3.7427022457122803, 0.5065156817436218, -0.3018874228000641, 0.9412669539451599, -0.6344384551048279, 0.23012495040893555, 0.2563542127609253, 0.34923669695854187, -2.00522518157959, -0.6754410266876221, 0.019401758909225464, -0.11917203664779663, 0.036024365574121475, 0.5600039958953857, 0.13263748586177826, 0.08227882534265518, -0.7996744513511658, -2.1981289386749268, 1.8988810777664185, 0.00016932534344960004, 0.5851700901985168, -0.5565246939659119, -0.004368417430669069, 0.24277228116989136, 0.2774882912635803, 1.0503827333450317, -0.655285656452179, 0.5353249907493591, 0.012190086767077446, -0.0133806262165308, 0.1550569087266922, 1.2650786638259888, -0.6599853038787842, 1.3885061740875244, 0.12991277873516083, -0.5407291054725647, -1.2263416051864624, 0.03744573891162872, 0.1834525763988495, 0.2964213788509369, -0.6964946985244751, -0.665212869644165, -0.4816579222679138, 0.16630743443965912, -0.05195190757513046, 0.38306063413619995, 0.2877492904663086, 0.04114983603358269, 0.0102216312661767, 0.7385336756706238, 0.3868173360824585, 0.014968426898121834, 1.039435863494873, -0.10931148380041122, -0.5369065999984741, -0.1334579735994339, 0.629176914691925, -0.10982456803321838, 0.20353391766548157, 0.7598729133605957, 0.37217679619789124, -0.6863880753517151, -1.013913869857788, 0.8735185861587524, 0.6792991161346436, -2.186413526535034, -0.4512270987033844, -0.05110664293169975, 0.6618570685386658, -0.04188475385308266, 0.6888824105262756, -0.09861336648464203, -0.4203805923461914, -0.23239834606647491, -0.619415283203125, 0.06015477702021599, 1.1828569173812866, 0.8096439838409424, -0.9934006929397583, -3.19004487991333, -0.4457233250141144, -0.05290304124355316, -0.07920406013727188, 0.01803814433515072, 0.13776585459709167, 0.04142330586910248, -0.25375697016716003, 1.9214164018630981, -0.12855322659015656, 0.33796635270118713, 0.1437317281961441, -0.21137754619121552, -0.09686058759689331, 0.1272728443145752, -0.5706151127815247, -0.376369446516037, 0.7182835340499878, -2.1172330379486084, 1.1297736167907715, 0.05605030804872513, -0.45575273036956787, 0.6523664593696594, 0.14751119911670685, -0.2657155692577362, -0.24763210117816925, -0.4293172359466553, -0.04734829440712929, -1.0688636302947998, 0.8886926770210266, 2.933471918106079, -0.5771886110305786, 0.06402499228715897, -4.532963275909424, 0.343325674533844, 0.0019063251093029976, 1.0033169984817505, 0.7173982858657837, -0.12702970206737518, -0.8752263188362122, 0.2984398603439331, -1.702106237411499, 0.1290305107831955, 0.23777830600738525, 0.3804894983768463, -0.02399512007832527, 0.11597544699907303, -0.6751528978347778, 0.10639889538288116, -1.1753759384155273, -0.9395633339881897, 0.670151948928833, 0.5936024785041809, 0.05078154429793358, 1.8951945304870605, -0.07994363456964493, -0.6008684039115906, -0.015136891975998878, -0.6383918523788452, 0.29112645983695984, -1.1151643991470337, -0.5566125512123108, 0.015238886699080467, -0.8887776732444763, 1.3348981142044067, -0.5528073906898499, -0.06947671622037888, 0.5382058024406433, -0.7360443472862244, -0.06184352934360504, -1.3907084465026855, -0.8486592173576355, -0.24673107266426086, -0.1602301150560379, 0.03566431626677513, 0.08279968053102493, 3.387561321258545, -0.7310127019882202, 1.4346954822540283, -0.49063974618911743, 0.19138497114181519, 8.704760551452637, 0.3665645122528076, -0.049746349453926086, -0.374671995639801, 0.31602737307548523, -0.14157262444496155, -0.8030731678009033, -0.6936540603637695, -0.5662529468536377, 0.2750789523124695, -0.23077785968780518, -2.010653257369995, -0.45391619205474854, -0.2055739462375641, -0.8916285037994385, -0.35021674633026123, 0.021242879331111908, 0.06062471866607666, 0.544987142086029, -0.2423759549856186, 1.2416470050811768, 0.016955744475126266, 0.9489895701408386, 0.0308332871645689, -0.16922879219055176, 2.98441743850708, 0.43409162759780884, 0.09191006422042847, -0.0036182913463562727, 0.047090597450733185, 0.6083812713623047, -1.7320713996887207, -0.5124514698982239, 1.1620360612869263, 0.43346551060676575, -0.16873757541179657, 1.9857584238052368, 1.250276803970337, 0.10109655559062958, -1.0011422634124756, 0.09196402877569199, -1.2556397914886475, -0.05452241748571396, -0.0771019235253334, 0.5514225959777832, -0.5454242825508118, 0.11797438561916351, -0.38715434074401855, -0.22871118783950806, -0.29275134205818176, 0.0374576598405838, 0.33562400937080383, 0.2629973590373993, -0.014213948510587215, 0.9310020804405212, 0.14628449082374573, 0.16990916430950165, -0.1759364902973175, 0.6369118094444275, 1.4473209381103516, 0.33737581968307495, -0.08331911265850067, 0.4507773518562317, 0.3334997892379761, 1.3908425569534302, -1.1218173503875732, -0.9001018404960632, -0.1390836238861084, 0.840778648853302, 0.5285717844963074, 0.03715699538588524, 0.1481449455022812, -0.016904722899198532, -0.10362425446510315, 1.13079833984375, 0.04223068431019783, 0.8755422234535217, 0.2988782823085785, -0.1076347827911377, -0.19429732859134674, -0.3713642358779907, 0.9419577717781067, 0.44465744495391846, -0.8303701877593994, 1.3902064561843872, -0.017534596845507622, -0.3878217339515686, -0.11933863162994385, -1.2315633296966553, 0.05496179684996605, 3.9190101623535156, -0.1610821783542633, -0.13867230713367462, -1.0605311393737793, -0.526074230670929, 1.1445846557617188, -1.7180343866348267, 0.2719760537147522, -0.47319796681404114, 1.4987236261367798, -0.8106085658073425, -2.3065853118896484, -2.3775014877319336, -0.20407269895076752, 0.3915889859199524, 0.9997512698173523, 0.05337165296077728, 0.34117648005485535, -1.1006979942321777, -0.3216189742088318, 0.6432791352272034, -0.1170138269662857, -0.20253834128379822, -0.15470635890960693, -0.12907898426055908, 0.021186912432312965, -0.5496408939361572, 0.671025276184082, -0.06587530672550201, -0.1970463991165161, 0.21096287667751312, 0.20189985632896423, -0.21092894673347473, 0.22393286228179932, 0.8307900428771973, 2.598759651184082, -0.53904128074646, -0.027723178267478943, -0.14912556111812592, 0.1258348971605301, -0.8712979555130005, -0.7375295758247375, -0.24516606330871582, 0.1459658145904541, -0.026387352496385574, -0.007229345850646496, -0.014181033708155155, -2.633368968963623, -0.630096435546875, 0.3403979241847992, -0.20765919983386993, 0.12254779785871506, 3.689420223236084, -0.030303360894322395, 1.2445931434631348, -0.3693925738334656, 0.37671852111816406, -0.8870387077331543, 0.12198155373334885, 0.767536997795105, -0.7203356027603149, -0.6531227231025696, -0.6475141048431396, -0.04632899537682533, -0.32923343777656555, -0.17859531939029694, 0.6453626751899719, 1.6786327362060547, 4.604522228240967, 0.016424285247921944, 0.5982674956321716, -2.0435452461242676, 2.2439310550689697, 1.0406513214111328, 4.5711750984191895, -0.5591069459915161, -0.174330472946167, 0.14930982887744904, -1.0141435861587524, -0.05649252235889435, 0.32787078619003296, -1.9964807033538818, -0.32169124484062195, 0.21126462519168854, -1.0294495820999146, -1.115136981010437, 0.017152568325400352, 2.813979387283325, 1.6447720527648926, 0.3568209111690521, 0.023029135540127754, -0.2662878632545471, 0.22103026509284973, 0.8366459012031555, 1.6244553327560425, -0.5477412939071655, 1.319462776184082, 0.10815868526697159, 4.792483806610107, -1.077665090560913, -1.1637424230575562, -0.2439374029636383, 0.008519671857357025, 1.9954273700714111, -0.13253846764564514, 0.5032757520675659, 0.5475408434867859, -0.04722421616315842, -0.7382913827896118, -0.1488446593284607, 0.46571171283721924, -0.5209957957267761, -0.8812953233718872, 0.18966259062290192, 0.7581139206886292, 0.16167563199996948, -0.16899342834949493, 0.8939195275306702, 0.010681433603167534, -0.16056211292743683, -0.3365631401538849, 0.5191525816917419, 0.0037388126365840435, -1.3010802268981934, -0.030777707695961, 0.33686137199401855, 0.22272546589374542, 0.07340335845947266, 2.00138783454895, -0.07526939362287521, -1.223609447479248, -0.8107650279998779, -1.3377020359039307, -0.9017224907875061, 0.32778677344322205, -0.044227417558431625, 0.00823764968663454, 0.10957465320825577, 0.20530171692371368, 2.137282133102417, 0.051537852734327316, -0.05879877135157585, -2.461179256439209, 0.027899714186787605, -1.5937939882278442, -0.045499976724386215, 0.8086938858032227, 2.129420757293701, 0.4252380132675171, 0.03335284814238548, -1.4714871644973755, -0.5393077731132507, -0.4897521734237671, 0.0016725168097764254, 0.05597158521413803, -0.06663738191127777, -0.021083123981952667, -0.5019561648368835, 0.31520822644233704, 0.3797726631164551, -2.390324592590332, 0.5971333384513855, -2.358048439025879, 0.5555440783500671, 0.2557137608528137, 0.14512194693088531, 3.1461989879608154, -0.7908666133880615, -2.157876968383789, -1.318338394165039, 0.2113998681306839, 1.9826277494430542, -0.1268889158964157, 0.07317127287387848, -0.9065448641777039, 0.4186852276325226, 0.7250704765319824, -0.6729164123535156, -0.14167910814285278, 0.4524925649166107, 0.09130441397428513, 0.5562173128128052, -1.7621514797210693, -0.0235750712454319, 0.09056069701910019, -0.7647678852081299, -0.7241618037223816, 0.16396048665046692, -0.21651016175746918, 0.4699643552303314, -0.34256088733673096, 0.4575329124927521, -0.43211445212364197, 0.05384696647524834, 0.006659741513431072, -1.8980937004089355, 0.09984184056520462, -0.002135577378794551, -0.7557839751243591, 0.05913475900888443, 0.07653100788593292, -1.41032874584198, 2.4518022537231445, -0.1555553525686264, -0.8787143230438232, 0.002086975844576955, -0.3901706039905548, 1.3728610277175903, 0.11117932945489883, -1.8086261749267578, 2.6583893299102783, -3.0095527172088623, -0.2919759154319763, 0.04200858622789383, 0.42580825090408325, 0.2050512731075287, -0.18244211375713348, 0.254271924495697, -0.8945198655128479, -0.07631373405456543, 2.191772699356079, 0.030264711007475853, 0.011207967065274715, 0.0413258858025074, 0.41204705834388733, 0.48265448212623596, -0.08940832316875458, 0.6794958710670471, -1.4493380784988403, -0.8762762546539307, 0.2717322111129761, -1.7951890230178833, -0.7929039597511292, -1.1585910320281982, 0.2824431359767914, 0.04659876972436905, -0.09507615864276886, -0.7496218681335449, -1.030490756034851, -0.6761823892593384, 0.01825946383178234, 0.37067076563835144, -0.030229486525058746, -1.236590027809143, -0.4922425448894501, 1.7496798038482666, -0.010407831519842148, -5.291274070739746, -0.007822846062481403, -0.07401517033576965, 1.5696494579315186, -0.09311649203300476, -0.7524715065956116, -1.2912406921386719, 0.017550991848111153, -0.2856910228729248, 0.9606095552444458, -0.16777567565441132, 0.45482057332992554, 0.4186771810054779, -0.0567283108830452, -0.2543570101261139, -0.20501990616321564, -0.21382898092269897, -0.3457370400428772, -0.13478955626487732, 1.2640589475631714, -0.045346274971961975, -0.36814284324645996, -1.7586091756820679, -1.6202270984649658, -0.016260191798210144, 1.097480297088623, 0.24870134890079498, 1.681143879890442, -0.4308409094810486, 0.9814590215682983, 0.9121366739273071, -0.25389862060546875, 2.642777681350708, -0.7640306949615479, -0.22021368145942688, -0.5131261348724365, 0.13265514373779297, 0.010382010601460934, 0.0013811280950903893, -2.7127084732055664, -1.653063416481018, 1.487457036972046, 0.0029504362028092146, 0.6089525818824768, 1.1838901042938232, -0.8794101476669312, -0.5058676600456238, 0.030106570571660995, -1.1743654012680054, -0.22459885478019714, -0.07587244361639023, -0.6581612229347229, 0.8963855504989624, 0.9137572050094604, 0.3145993947982788, 0.4455200135707855, 1.1177135705947876, 1.8524177074432373, 0.44480374455451965, -0.0014319992624223232, -0.16739852726459503, 0.49468034505844116, -0.08789074420928955, -1.140785574913025, 0.19453679025173187, -0.1504264920949936, 0.429024875164032, 0.5494911670684814, -0.013059031218290329, -0.004900780506432056, -0.06131720170378685, 0.3654068410396576, -0.17232747375965118, 0.37133312225341797, -2.685349225997925, -0.3377281427383423, 0.745265543460846, -1.761677622795105, 0.20559056103229523, -0.032805148512125015, -1.04530930519104, -0.5294184684753418, 0.6517459154129028, -0.041041601449251175, 0.6097171306610107, -0.5092530846595764, -0.09038320183753967, 0.037373024970293045, -0.39274656772613525, -0.3665469288825989, 0.18669907748699188, -0.306543231010437, 0.33790794014930725, 1.0926198959350586, 0.010792579501867294, 0.11002551019191742, 1.2498092651367188, -0.25066471099853516, 0.09389486163854599, 0.9763647317886353, -3.184539318084717, 0.15111318230628967, 0.01829274371266365, -0.4947134554386139, 0.022726429626345634, 0.5157093405723572, 0.24542640149593353, -0.05479442700743675, 1.5093461275100708, 0.35162240266799927, -0.14603322744369507, -0.09872577339410782, -0.7640528082847595, 0.46473413705825806, 0.0416705496609211, 0.583564281463623, -0.5661727786064148, 0.0358414463698864, 0.5142489671707153, -2.3156843185424805, -2.5646474361419678, -2.312056303024292, -0.4306947886943817, 0.9382100701332092, -0.1123478040099144, -0.01998152583837509, -0.26370155811309814, -0.006527377292513847, 0.5123540759086609, 0.7817871570587158, 2.2186176776885986, -5.794219970703125, 0.13242454826831818, -0.07332803308963776, -0.2037418931722641, -0.14646930992603302, 2.142658233642578, 0.6276606321334839, 2.7609550952911377, -0.05600491911172867, -0.14487770199775696, 0.36724215745925903, 0.9416752457618713, 4.132735729217529, -0.041283223778009415, -0.026716938242316246, -0.8543791174888611, 0.7324461340904236, -1.0598855018615723, -0.15841327607631683, -0.5249288082122803, -0.12186720222234726, -0.9885938167572021, 0.8813965916633606, 0.3252353072166443, -0.023995783179998398, -0.11391375958919525, 0.11787377297878265, -1.7396328449249268, 0.08838392794132233, -0.051309939473867416, -0.09337815642356873, 0.2930721342563629, 0.2510239779949188, 0.5177528262138367, -0.14353640377521515, -0.010403466410934925, 0.4477755129337311, 0.5342172384262085, 0.13027197122573853, -0.2022455334663391, -1.356040596961975, -0.4959695041179657, 1.6788876056671143, 0.08590424805879593, -1.1376051902770996, 0.8993698954582214, -0.15311908721923828, -0.0193976778537035, 1.001708984375, -0.03575940057635307, 0.2490941435098648, -1.3650931119918823, -0.2221851348876953], [-0.22828364372253418, -2.0500617027282715, 0.01940232887864113, -1.5541247129440308, 0.25906795263290405, -1.1969821453094482, 0.006545383483171463, -0.32521727681159973, -0.7465859055519104, 1.5634090900421143, -1.5296000242233276, 0.1287483125925064, -1.2962615489959717, -0.35032328963279724, 0.5335744023323059, -1.5723899602890015, -0.26816120743751526, -0.09970877319574356, 0.09860648959875107, -0.3279288411140442, -0.6171212792396545, 0.40252920985221863, -0.03517133742570877, -1.4990280866622925, 0.27204111218452454, 0.9201473593711853, -0.35394287109375, -0.29215484857559204, 0.6346442699432373, 0.020370110869407654, -0.5073972940444946, 0.3960622251033783, 0.133852019906044, 0.3320065438747406, 0.01814749464392662, -0.006507169920951128, 0.04245191439986229, 0.01643957383930683, 0.3448578715324402, 0.6706975102424622, 0.25272953510284424, 0.5425779223442078, -0.22787821292877197, -0.013427509926259518, -0.07522620260715485, 1.1441302299499512, 0.4437773525714874, 0.39827439188957214, 0.8990865349769592, -0.5623988509178162, -0.4149973392486572, 0.07582160085439682, 0.03475739061832428, -0.2200910449028015, 0.014141357503831387, 1.7106375694274902, 0.020696479827165604, -0.002219305606558919, 0.008214431814849377, 0.0160977803170681, -0.0298940297216177, 0.1766023337841034, 0.00527009554207325, -0.99226975440979, -1.4490275382995605, 0.15279532968997955, -1.029590129852295, 2.7507569789886475, -0.33143121004104614, 1.7092254161834717, 0.08547099679708481, -1.398235559463501, 1.1348086595535278, -1.2441942691802979, 0.3773525655269623, -0.5277047157287598, 0.07693679630756378, 0.796855092048645, -0.061874572187662125, 0.10569100826978683, -1.5300967693328857, 0.11823257058858871, -0.10809578001499176, -0.12087907642126083, -0.07691022753715515, 0.39753368496894836, 0.02137543074786663, 0.24990281462669373, -1.1296396255493164, 2.073423385620117, -0.07226665318012238, 0.32539501786231995, 0.05453319475054741, 0.9675600528717041, -1.0773983001708984, -0.10283885896205902, 0.0645563155412674, 0.32828205823898315, -1.3774845600128174, 0.22998660802841187, -0.0051880888640880585, 1.1786160469055176, -1.076141119003296, 0.1458204984664917, -0.0018019919516518712, -0.031398218125104904, 0.07104992121458054, 0.013672078028321266, -1.6416817903518677, 1.106245994567871, -1.8203262090682983, 0.0056945341639220715, 0.3947008550167084, 0.0465315543115139, 0.30547043681144714, 1.4173617362976074, 0.4696233570575714, -1.4716674089431763, -0.36123645305633545, 1.347273826599121, 0.4042486548423767, -2.541250228881836, 0.07185559719800949, 1.77333664894104, 0.03447125852108002, -0.27993345260620117, -1.1826001405715942, -0.015870552510023117, -0.6883168816566467, -0.307888001203537, 0.04261403530836105, -2.5065197944641113, -0.053372547030448914, 0.2588161528110504, -0.8442116379737854, -0.6487365365028381, 0.38188081979751587, -0.3802265524864197, -0.30712777376174927, -0.5343377590179443, -0.4624955952167511, 0.240503191947937, 0.0718206986784935, -0.22495827078819275, 1.7713327407836914, 0.06183348596096039, 0.8575347065925598, 0.9836503863334656, -0.004506675060838461, 0.01029417384415865, 0.7332066893577576, -0.40561890602111816, -0.10433299839496613, -0.11737001687288284, 1.2818002700805664, -0.6078054308891296, 0.4065023958683014, 0.03491784259676933, -0.35405564308166504, 1.539230465888977, 0.0005032189656049013, -0.006350488401949406, 0.14031679928302765, -1.1917668581008911, -0.2057460993528366, 0.017443297430872917, -0.49370548129081726, -0.45190179347991943, -0.3550563454627991, 0.11864177137613297, -1.0269216299057007, 0.28718259930610657, 0.5500547885894775, -2.7875397205352783, 0.2872445285320282, -0.1824672818183899, 0.054425060749053955, 1.0676133632659912, -0.25291985273361206, -0.12201620638370514, -0.9949510097503662, -0.03049967624247074, 0.016818802803754807, 0.6352847218513489, 0.7711178660392761, -0.5265129804611206, 0.026960236951708794, 0.020622391253709793, -0.08828030526638031, 2.3606200218200684, 0.19124053418636322, 1.3937785625457764, 0.6055822968482971, 2.155750036239624, -0.16619937121868134, 0.06538046151399612, -1.5134353637695312, 0.0015045471955090761, 1.3256596326828003, -1.9036383628845215, -0.08954542130231857, -0.14496339857578278, -0.08272847533226013, 0.0027156982105225325, -0.4776242971420288, 0.18393996357917786, -2.069255828857422, 1.7857531309127808, -0.9484332799911499, 0.21510359644889832, -0.049650486558675766, -0.027823902666568756, -0.05825084075331688, -0.004372967407107353, 0.34684181213378906, 1.484628677368164, 1.389830231666565, 0.4239266514778137, -0.050514742732048035, 0.5346598625183105, 0.1826750785112381, -0.13425849378108978, 0.7520920038223267, 2.2286183834075928, 0.674212634563446, -0.1162494346499443, -1.9897887706756592, -0.0518917590379715, -0.40280866622924805, -0.05693644657731056, -0.28135982155799866, -0.010152608156204224, 0.2274002581834793, -0.3420960307121277, 0.5915700197219849, -0.49542638659477234, -0.2551156282424927, 0.24088430404663086, -0.21917229890823364, -0.0503276064991951, -1.7668707370758057, 0.3258499801158905, 2.6655185222625732, -0.8069647550582886, -1.1121389865875244, 0.1662372499704361, 1.8212378025054932, -3.143686294555664, 0.27390697598457336, -0.03933709114789963, -0.14964339137077332, 0.6053998470306396, 0.023329811170697212, -0.09335849434137344, 1.391784429550171, 0.14112763106822968, 1.6403899192810059, -0.06794750690460205, -0.0365387462079525, 0.1094958633184433, 0.43081873655319214, 0.11048736423254013, -0.7644504308700562, -0.29406243562698364, -0.08700966089963913, 0.41316574811935425, 0.845439076423645, 0.012158920057117939, 1.5074504613876343, -2.4648945331573486, -0.0007313982350751758, -1.038456678390503, -0.6444617509841919, 1.746111512184143, 0.5837640762329102, 0.3513771891593933, 0.8048818707466125, -0.15537084639072418, 1.0729669332504272, 1.7597193717956543, -0.012775393202900887, 0.757187008857727, 1.1249604225158691, -0.03929600492119789, -0.6588005423545837, -1.1078604459762573, -0.5714657306671143, 1.93504798412323, 0.1883801370859146, -0.5891919136047363, -0.19152933359146118, -0.22810208797454834, 0.009059439413249493, 1.6407831907272339, -0.20773476362228394, 0.44232213497161865, -0.046201590448617935, 2.6263325214385986, -0.9522199630737305, 0.05679699406027794, -0.18953338265419006, 0.01599884033203125, 0.019143691286444664, 0.27738362550735474, -3.5044240951538086, 0.953593373298645, 0.05578651279211044, 1.8536471128463745, 0.4209481477737427, 0.07359596341848373, -0.007238021120429039, 1.1361175775527954, -0.23846277594566345, -0.529106616973877, 0.20222067832946777, -0.6158210635185242, -0.66379714012146, -0.37533295154571533, 0.8266739845275879, 0.18725629150867462, -0.024372288957238197, 1.4612741470336914, 0.34643375873565674, -0.6016668677330017, 0.10249428451061249, 0.41542837023735046, -0.2462390959262848, 5.299616813659668, 0.9825930595397949, 2.463703155517578, 0.5286856889724731, 0.3738386631011963, 0.13447895646095276, -0.014096890576183796, -0.35124140977859497, 0.26872384548187256, -1.275019884109497, 2.0479838848114014, 0.023673323914408684, 0.006018185522407293, -1.7397090196609497, 0.5525236129760742, -0.5136645436286926, 0.8589950799942017, 1.6822128295898438, 0.22260025143623352, 0.4647836983203888, 0.17866039276123047, -0.7010143399238586, 0.22779680788516998, 0.5807269215583801, -0.34320399165153503, -1.2363810539245605, -0.18188002705574036, 0.07120021432638168, 0.48944222927093506, -0.1269458532333374, 1.2740062475204468, -5.172110080718994, -0.16945555806159973, 1.0276566743850708, -0.6696134805679321, 0.06311502307653427, -0.32190176844596863, 0.27682191133499146, 1.9756031036376953, -1.3151209354400635, 0.6116119623184204, 0.25762516260147095, 1.9188910722732544, 0.10983393341302872, -0.22227205336093903, -0.18002478778362274, 1.5514662265777588, 0.0790904313325882, -1.3874074220657349, -0.98785400390625, -0.02210867777466774, 0.9115571975708008, 0.6843340992927551, 0.5012290477752686, 0.11267012357711792, -0.02715589851140976, -0.972322404384613, 1.4506111145019531, -0.1409866362810135, 0.14223434031009674, 0.008070209994912148, 1.3635741472244263, 0.4558413326740265, 0.9071186780929565, 0.01407708041369915, 0.05383339896798134, 0.04483901709318161, 0.09021448343992233, -0.3298173248767853, -0.25201356410980225, -0.09917408972978592, -0.5157043933868408, -0.00042650956311263144, 0.624559760093689, -0.16661278903484344, 0.034314289689064026, -0.4608501195907593, 0.01916276104748249, 0.9352369904518127, -0.6324147582054138, 0.13455352187156677, 1.718939185142517, 1.0040647983551025, -1.4481582641601562, 0.27725401520729065, -1.4380162954330444, 0.15389898419380188, 0.03740565851330757, 0.173301562666893, 0.26081550121307373, 0.31906431913375854, -0.5077484846115112, -0.9419530630111694, 0.0377739742398262, 1.4076777696609497, 0.19195176661014557, -0.6785692572593689, -0.6469916701316833, 0.8956639170646667, 0.007991691119968891, 0.6042065620422363, -0.5730175971984863, 0.3530206084251404, 0.029513554647564888, -0.9359306693077087, 0.2517034709453583, -1.0966802835464478, 0.7490184307098389, 1.4692429304122925, -0.05828295648097992, 0.07154744863510132, 0.2830863893032074, 0.45813068747520447, -2.193681478500366, 0.2281927615404129, 0.05069950968027115, 0.02206132747232914, 0.5787136554718018, 1.5214446783065796, 0.8021268844604492, 1.4921447038650513, -1.078261137008667, 0.011540056206285954, 0.1680394411087036, -1.0859829187393188, -0.006855027750134468, -0.21767060458660126, -1.2333953380584717, 0.6351419687271118, 0.2734161615371704, 0.5820581316947937, -2.9709830284118652, 0.041596245020627975, -0.13857653737068176, 0.43014898896217346, -0.014925309456884861, -0.13564716279506683, -0.09378676861524582, -0.24244099855422974, 2.6646652221679688, -0.15947183966636658, 1.0062885284423828, -0.0035285584162920713, 2.6367533206939697, -0.9385848045349121, -0.5099669098854065, -1.5788439512252808, 0.1680561751127243, 0.3675926625728607, 0.18330501019954681, -2.897907018661499, -0.4519808888435364, 0.03480684384703636, -0.1272837519645691, -0.9968921542167664, 0.22683094441890717, 0.049650393426418304, -0.0471821092069149, -1.091639757156372, -0.27201831340789795, 0.9668114185333252, 0.344367116689682, -0.001632080296985805, 0.6503377556800842, 0.42752805352211, 0.6610766053199768, 1.9978638887405396, 0.32124802470207214, 0.14077337086200714, 1.2869434356689453, -0.2118438482284546, -0.5040996670722961, 0.09234517067670822, 8.974692344665527, 0.444225013256073, -0.23770301043987274, -0.06716162711381912, 1.5371885299682617, -0.022127671167254448, -0.531924307346344, 0.1173715591430664, -0.04880276694893837, -0.39756742119789124, 0.7560551762580872, 0.03671770170331001, 0.07935799658298492, 1.6348745822906494, -1.441070556640625, -1.002510905265808, -0.9776818156242371, -1.5471528768539429, 1.3004305362701416, -0.5871875286102295, 1.0362248420715332, -0.27836874127388, 0.07810725271701813, 0.1432742029428482, -0.07676760107278824, 0.6232119798660278, -0.03548251464962959, -0.16280502080917358, -0.3624788224697113, -0.9229512214660645, -0.06536900997161865, 1.030605435371399, -0.026785731315612793, -0.3585869371891022, -2.5918447971343994, -1.7404446601867676, -0.7843406200408936, 0.11332447081804276, -1.078041672706604, 0.5659612417221069, -0.32714393734931946, 0.007744681090116501, 0.5832033157348633, -0.15509958565235138, -0.1743626743555069, -0.03850584104657173, -0.5507640838623047, -0.38342466950416565, -0.07701040059328079, -0.0002685383951757103, 0.08166119456291199, 0.029329614713788033, -0.7954462766647339, -0.0862085148692131, 1.4374213218688965, -0.819673478603363, 0.16716064512729645, 0.47786128520965576, -0.12157481908798218, -0.015418834052979946, -0.8413681983947754, -1.6321702003479004, -0.15494278073310852, 0.07646217942237854, 3.304643154144287, 2.198848009109497, -0.019564393907785416, 0.3853508532047272, 2.7173588275909424, 0.8461663126945496, 0.06350789219141006, 1.1487025022506714, -0.5016783475875854, 0.08662158995866776, 0.3262269198894501, 0.9608558416366577, 0.6606281399726868, 0.6274730563163757, -1.4156461954116821, -0.5889215469360352, 0.3963525891304016, 0.3062257170677185, -0.17448803782463074, 0.02488424815237522, 0.11478495597839355, -1.2710260152816772, 0.06887511909008026, 0.7607129812240601, -0.3162069022655487, -0.10160928219556808, -0.8606595993041992, 0.044529400765895844, -0.3743591904640198, -0.2847824692726135, -0.01842944510281086, 0.036134690046310425, 0.9697774052619934, -0.6033309698104858, 0.5389429926872253, 0.014883534051477909, 0.14322608709335327, 0.08552537858486176, -0.10212505608797073, 0.3351783752441406, -0.01897490583360195, 0.015785962343215942, -0.30922383069992065, -0.3206821084022522, -2.9933109283447266, 2.728980779647827, -1.2722406387329102, -0.12922851741313934, -1.263268232345581, -0.7972477078437805, -0.327532023191452, 0.5261114835739136, 0.01534448005259037, 0.6473435759544373, 0.049919623881578445, 2.545475721359253, 0.003834993578493595, -0.06675243377685547, -0.42197147011756897, -0.6329020261764526, -0.018317734822630882, 0.21695367991924286, 1.3946077823638916, -0.055871229618787766, -0.5112783312797546, -0.01998979225754738, 0.9718561172485352, -0.37556836009025574, -0.0004958455683663487, -0.3175569474697113, -0.9738432765007019, 0.13568317890167236, 0.28085586428642273, 0.07639020681381226, -0.026712141931056976, 2.85408353805542, -0.5969332456588745, -0.3067920207977295, 5.078501224517822, -0.6759803295135498, 0.08321940153837204, -1.0955514907836914, 0.9416444301605225, -0.007456707768142223, 3.9466235637664795, 0.20892253518104553, -0.9870990514755249, 1.9973593950271606, 0.07732599973678589, 1.2924010753631592, 0.33272984623908997, -0.12240267544984818, 0.0018247427651658654, -0.5308443903923035, 1.1370744705200195, 0.18572767078876495, 0.1367879956960678, -0.04260418191552162, 0.11821316927671432, -0.09276510030031204, -0.9579668641090393, 0.08157425373792648, 1.8666144609451294, 0.5698034167289734, 0.4970567226409912, -0.06323430687189102, -0.6438537240028381, 0.1916072815656662, 0.6063202023506165, -1.6902624368667603, -1.7034308910369873, 3.3108325004577637, 0.0813835859298706, -0.2008964717388153, 1.8106341361999512, -0.04365580528974533, -0.01743803359568119, -0.14921149611473083, 0.5772098898887634, -0.6767613291740417, -0.28767073154449463, -0.17271879315376282, 0.16750001907348633, -0.37696385383605957, 2.2644712924957275, 0.11618677526712418, 1.0049610137939453, -0.8186240792274475, 0.4385618269443512, 0.6664400696754456, -0.06322270631790161, 1.569392442703247, 0.14846724271774292, 0.035758040845394135, -1.0650391578674316, -1.9102033376693726, 0.031357184052467346, 0.461531400680542, 0.3834327459335327, -0.09294334053993225, -0.2383660078048706, -1.7580986022949219, 0.09551108628511429, -0.15068753063678741, -0.5590022802352905, 1.247179627418518, 0.4980712831020355, 0.46637997031211853, 1.1491830348968506, 0.5951469540596008, 0.10649171471595764, -0.1549740433692932, 0.0037477074656635523, -0.09681597352027893, -0.4353526532649994, 0.1532866358757019, 2.0146117210388184, -0.17092829942703247, -4.332618236541748, 1.7865747213363647, -1.2141056060791016, 0.3138144910335541, 0.03849097341299057, -0.560157299041748, -0.33684810996055603, 0.6003977060317993, -0.4169885218143463, 0.3581331968307495, 0.022470543161034584, -0.007723389193415642, -0.32147637009620667, 0.1339399367570877, -0.07834437489509583, -0.5352858901023865, 0.07903482019901276, -0.06761696934700012, 0.0014535682275891304, -0.8948138356208801, -1.6594021320343018, -0.13403074443340302, 0.25018900632858276, -0.42243966460227966, -5.8125242503592744e-05, 0.186506450176239, 1.1258772611618042, 0.5057980418205261, -0.5518785119056702, -0.3114745020866394, 1.036208987236023, -1.5329182147979736, 0.3869909346103668, -0.2410673052072525, 0.18393629789352417, -0.1334279328584671, 0.8309937119483948, -0.005811731331050396, 2.165630340576172, -0.07235650718212128, -0.034146834164857864, 0.15070153772830963, -0.07065246254205704, -0.0671977624297142, 0.36082035303115845, -0.26391202211380005, 0.08055923879146576, 0.9463050365447998, 0.03149116411805153, 1.321358561515808, 0.13549178838729858, 0.14857718348503113, -0.011732230894267559, -0.3753233551979065, -0.2105022370815277, 0.6316224336624146, -0.19324931502342224, -2.6172587871551514, 0.0005858715157955885, -0.03961004689335823, -0.05227568373084068, -0.056465696543455124, -0.0927640050649643, -0.004532998893409967, -1.9321101903915405, 0.5521726608276367, 0.4377917945384979, -0.8437244892120361, 0.3730390667915344, 0.02846413664519787, 0.14849422872066498, -0.6204820275306702, -0.003317014779895544, 0.3548455238342285, -0.12195524573326111, -0.09863049536943436, -0.1578599512577057, 0.2418735772371292, -0.35066884756088257, 0.0457368865609169, 0.1151922196149826, 0.003284197999164462, 0.13208366930484772, -0.6086775660514832, -0.5853535532951355, -0.763279914855957, 0.03557801991701126, -1.554844617843628, -0.41393399238586426, 0.0068101114593446255, -1.1780853271484375, -0.005013685207813978, 0.06493788957595825, -0.10711979866027832, -0.8174937963485718, 0.1271735280752182, -0.18283109366893768, -1.0306469202041626, -0.018399178981781006, 1.5625035762786865, 1.3097881078720093, -0.04792604595422745, -7.239501476287842, -0.793108344078064, 0.021524492651224136, 1.213858962059021, 0.12844562530517578, 0.6751610040664673, -1.0512616634368896, 0.028276151046156883, -0.7635825872421265, 0.41212794184684753, 0.8091344833374023, -0.5079627633094788, -0.9628455638885498, 0.5383515357971191, -1.6640217304229736, 0.3063785433769226, 3.3419337272644043, 0.5325716137886047, -0.6091268658638, -1.1547698974609375, -1.2260208129882812, -1.043036937713623, -0.9914348721504211, 0.2198135107755661, -0.02486984059214592, 0.36511942744255066, -0.033090777695178986, 1.3647325038909912, 0.3147018551826477, 0.10897259414196014, -0.12042268365621567, -0.2955330014228821, -0.27627086639404297, -0.23272544145584106, 0.006894960533827543, -1.5800913572311401, 0.5797855257987976, 0.14311476051807404, -4.00250768661499, -0.14649911224842072, 0.4032914638519287, -0.1339942216873169, -0.36403465270996094, 0.170322448015213, 0.9027690887451172, 0.31309810280799866, 0.308847576379776, -0.30581730604171753, 0.6963800191879272, -0.013206254690885544, 0.2386750429868698, -1.3193933963775635, 0.003074388485401869, 1.154003381729126, 0.8815523386001587, 0.37737125158309937, 0.1680373102426529, -0.7639103531837463, 0.8730912804603577, -0.03766011819243431, -0.016406042501330376, -0.24709492921829224, 0.19994021952152252, -0.20082442462444305, 0.16715365648269653, 2.1671030521392822, -0.09232912212610245, 1.598631739616394, 0.018930858001112938, -0.6175311803817749, -2.21856951713562, 0.4237722158432007, 0.09132815897464752, 0.35794690251350403, 0.08097635209560394, -2.0553276538848877, -0.5130923986434937, -0.09487901628017426, 1.136875033378601, -0.1437971442937851, -0.9074811339378357, -0.006027698516845703, -0.052248526364564896, 0.8272009491920471, -0.7290215492248535, -0.25510847568511963, 0.1940763294696808, 0.6210155487060547, 0.6729031801223755, -0.12249177694320679, 0.8797606825828552, -0.12960022687911987, 0.5775318741798401, 0.04701978340744972, -0.03440263494849205, 0.08268602192401886, -0.2862999737262726, 0.013505621813237667, 0.8365983963012695, -0.2582991421222687, -0.7159949541091919, -0.22178144752979279, -0.42229077219963074, -0.16773535311222076, -0.5322317481040955, -0.18704257905483246, 0.09219411760568619, 3.025404930114746, -0.3011450469493866, 2.300273895263672, -0.28691357374191284, -0.7540408968925476, -0.07573091238737106, 0.967251718044281, -0.7400223016738892, -0.04408831149339676, 1.5798319578170776, -1.4261013269424438, -0.6272040009498596, 2.1234636306762695, 0.3163922429084778, 0.023927487432956696, 0.07616683095693588, 2.746187448501587, -0.7350117564201355, -0.8791818022727966, -0.6651346683502197, -1.1956831216812134, 0.3046063780784607, 0.09680241346359253, 0.2371884435415268, -0.30336686968803406, 1.9941486120224, 1.8376996517181396, -0.5096561908721924, 0.21426871418952942, 0.48420703411102295, -1.293595314025879, 0.8797225952148438, -0.35487625002861023, 2.377241849899292, 0.9998633861541748, -1.7748078107833862, 0.04604581370949745, -0.5043702721595764, 0.8470123410224915, -1.3572895526885986, -3.580077648162842, -0.2791224718093872, -0.015072152949869633, 0.4964637756347656, -0.31612658500671387, 0.2710266709327698, 0.2858291566371918, -3.0927250385284424, 0.036202527582645416, 0.4613454341888428, -1.3182231187820435, 0.4560900628566742, -0.02449321188032627, -0.18763159215450287, -0.7089701294898987, -0.050779569894075394, -0.030004557222127914, -0.017585396766662598, 0.4697262644767761, 0.786556601524353, 1.8496713638305664, -0.15173405408859253, 1.8190406560897827, 0.013444988988339901, 0.18976357579231262, 0.09826744347810745, 0.2340376377105713, -0.39383748173713684, 2.297579050064087, -1.6911660432815552, -0.6569990515708923, -0.6311272382736206, -0.8269954919815063, 0.023127399384975433, -1.3047945499420166, 0.689856767654419, -1.0868381261825562, -0.14183136820793152, -0.10801434516906738, 0.06219252198934555, -0.06812085211277008, 0.43727344274520874, 0.5287973284721375, -0.45919641852378845, -0.3905392289161682, 0.25861692428588867, 1.1930242776870728, -0.21542713046073914, 0.7020303606987, -1.696319580078125, -3.4727084636688232, -0.5137268304824829, -0.0007006783271208405, 0.1952403038740158, -0.03739337995648384, 0.5221323370933533, 0.26870349049568176, -0.02734456956386566, 0.40382063388824463, 0.398167222738266, -0.509691059589386, 0.2834620177745819, 0.1158575639128685, -1.7026119232177734, -2.7339658737182617, -0.6033809185028076, 0.7472071051597595, -0.4351707696914673, -0.03257714584469795, 0.33059224486351013, -0.28156018257141113, 0.024335846304893494, 0.5697216987609863, -0.1060437336564064, 0.205659419298172, -0.5353602170944214, -0.03619462251663208, -0.31593623757362366, -1.7247803211212158, 0.32151156663894653, 0.09727557748556137, -0.07860149443149567, 0.2403886318206787, 0.1633196473121643, -0.404083251953125, 0.7744550108909607, 0.021148737519979477, 0.5928403735160828, -0.8156255483627319, -1.9744755029678345, 0.45316067337989807, -0.6880984902381897, -0.07673473656177521, 0.32685890793800354, -0.13453181087970734, -0.042153213173151016, -0.11760030686855316, 0.4177902340888977, 0.059365320950746536, -0.6107244491577148, 0.21032169461250305, -0.17438152432441711, 3.6969492435455322, -0.020141098648309708, 0.02721184678375721, 0.17310777306556702, -0.48254114389419556, 0.5607114434242249, -0.0017341687344014645, -2.926748752593994, 0.8154051899909973, -1.177586317062378, 0.41613081097602844, -0.6579465270042419, -0.1388767957687378, -1.539634346961975, -0.0220633614808321, -0.24200531840324402, 0.17227362096309662, -1.5509154796600342, 0.26070457696914673, -0.376770943403244, 1.1738083362579346, 0.00018465596076566726, -0.28313881158828735, 1.9002505540847778, -1.3011800050735474, 0.0010946240508928895, 0.1670447438955307, 1.4420942068099976, 0.1659204214811325, 0.08375416696071625, 0.3394618034362793, 0.5414872169494629, -0.5664835572242737, -2.001349449157715, -0.15443098545074463, -1.3420772552490234, -0.8340774774551392, -2.108603000640869, -0.03174392879009247, 0.15874481201171875, -0.6789957284927368, -2.482128620147705, -1.1611658334732056, 0.0005857569049112499, -0.7921351194381714, 0.263002872467041, -0.1360853612422943, -0.1791718602180481, 0.0951833575963974, 2.388303518295288, -0.7859448790550232, 0.7811551690101624, -0.4221922755241394, -0.3411581814289093, 0.22685644030570984, -1.3492827415466309, -0.5283289551734924, 0.05589865520596504, -0.00879010371863842, 0.1223997175693512, -0.13971222937107086, -0.5333084464073181, -0.1985466182231903, -0.38555532693862915, 0.07643096894025803, 0.28203415870666504, 0.9914897680282593, -1.0153794288635254, -0.7775905132293701, 0.0020211676601320505, 0.6524546146392822, -0.3020724356174469, -0.5119555592536926, 0.10080743581056595, -0.15717869997024536, 0.6071690320968628, 0.6698037981987, -0.7332597374916077, 0.5345556139945984, -0.3466818928718567, -0.24713917076587677, 0.46214649081230164, -0.33604827523231506, -3.9236559867858887, -0.1639755666255951, -0.17955303192138672, -0.2619039714336395, -0.28896138072013855, 1.7899549007415771, 1.0786644220352173, -0.02107137441635132, 0.07479259371757507, -0.2462615668773651, 4.834648132324219, 2.6075854301452637, -0.010129308328032494, 0.06628543883562088, -0.3313225507736206, -0.4101492166519165, -0.5996465086936951, 0.01972217671573162, -0.7827535271644592, -0.09416899085044861, -1.646901249885559, -0.015962891280651093, -0.35536301136016846, 0.10549426823854446, -0.13100534677505493, -0.12254256010055542, -1.0013189315795898, 0.519292950630188, -0.03158310428261757, -0.7666321396827698, -0.4015916883945465, 0.6618204116821289, 0.0035154332872480154, 0.23483708500862122, -0.3299161493778229, 0.02061799354851246, 1.0390371084213257, 0.012984064407646656, 0.6026791930198669, -0.00800532940775156, -0.08209466189146042, 0.2154199630022049, 0.2971033751964569, 0.048643212765455246, -0.0069063282571733, 1.588641881942749, 2.2774863243103027, 0.6678583025932312, 0.08155722916126251, -0.4591287672519684, 0.6052573919296265, -0.08734934777021408, 0.5052577257156372, 0.2688444256782532, -0.015513554215431213, 0.33706650137901306, -0.9410666227340698, 0.281522661447525, 0.1404540091753006, 0.30250921845436096, -0.2696763873100281, 0.5976670980453491, 0.5516982078552246, 0.5533432364463806, 0.029595432803034782, -0.33094900846481323, -0.15824826061725616, 0.08903906494379044, 0.08674322068691254, 1.1069867610931396, 0.5881829857826233, 0.004798868205398321, 0.5223957896232605, 1.2458676099777222, -0.22711880505084991, 1.6985490322113037, 1.0008853673934937, 0.05379438400268555, -0.006952186580747366, -2.651050090789795, -0.3568170666694641, -0.3252561092376709, -0.7144035696983337, -0.17152920365333557, -3.51509165763855, -0.6638587117195129, 0.3329007923603058, 0.4308598041534424, -1.0344839096069336, 0.16731129586696625, 0.11020319908857346, 1.1944148540496826, -0.32689353823661804, -0.0663985013961792, -0.8959510922431946, 0.6468783617019653, 1.1830631494522095, -0.8540298342704773, 0.014541580341756344, -0.8516674041748047, -0.0884685218334198, 0.22615700960159302, 0.20998403429985046, 0.2825147807598114, -0.01645192690193653, -0.24336501955986023, -2.0383715629577637, -0.0004453763540368527, -0.356523722410202, 1.3772858381271362, -0.010730496607720852, 0.05728830397129059, 0.9341771006584167, 0.04393354803323746, -0.283417671918869, -0.6503955721855164, 0.10511156916618347, 0.6018649339675903, -0.04429008811712265, 0.572446882724762, 1.3639590740203857, 0.036952223628759384, -1.8291658163070679, 0.7166441679000854, 0.964335560798645, -0.10151777416467667, -0.19707347452640533, -0.0748845785856247, -0.00939874816685915, 0.7116949558258057, 0.8572327494621277, -1.1572819948196411, -0.1673193722963333, -0.24525810778141022, -0.4675239026546478, 0.3761945366859436, 1.6616047620773315, 0.17969073355197906, 0.021266888827085495, 0.3902616798877716, 0.25119978189468384, 1.597618579864502, -0.2545376718044281, -0.986149787902832, 0.34533289074897766, 0.4925302565097809, 0.15747232735157013, -0.2006394863128662, -0.04805251210927963, -0.11887293308973312, -1.0556614398956299, -0.057336512953042984, -0.35472047328948975, -0.5491796135902405, 0.2783030569553375, -0.07349784672260284, 0.37052273750305176, -0.20733055472373962, -0.02873699739575386, 0.0036639596801251173, 0.002185042016208172, -0.011552547104656696, 2.0563995838165283, -1.3550610542297363, -0.2800949215888977, -0.5057154297828674, 1.0354645252227783, 1.5380890369415283, 0.4354074001312256, -0.9542833566665649, 0.9528285264968872, -0.20360729098320007, -0.014165663160383701, -0.7929863333702087, 0.5328680276870728, 1.1961181163787842, -0.04968534782528877, -0.055565692484378815, 0.17284001410007477, -1.092880129814148, -0.3699151277542114, -1.5260030031204224, -0.1434333175420761, 0.25746625661849976, -0.00505725247785449, -0.10483437031507492, -0.8693527579307556, -0.07272721827030182, -0.8428482413291931, -1.1662302017211914, 0.04242055118083954, -1.5817738771438599, 0.737259030342102, -0.39496809244155884, 0.3166883885860443, -0.4480624496936798, 0.14949363470077515, -0.6691238880157471, 1.0507900714874268, 0.7790424823760986, 2.6684224605560303, -0.05795628949999809, 1.6926963329315186, -0.5128039121627808, 0.6325591206550598, 0.02259601280093193, -0.0922037735581398, -0.055743128061294556, -0.45910289883613586, 1.9485502243041992, -0.885974109172821, -0.5738658308982849, -0.12189009040594101, 0.026082508265972137, -2.1879332065582275, 0.029674824327230453, -5.528634071350098, -0.003852616995573044, 0.33989834785461426, -2.06148624420166, -0.011213495396077633, 0.9190623164176941, -0.679327130317688, -0.11840982735157013, 0.008997540920972824, 0.7280126810073853, -0.21609929203987122, -0.1446644812822342, -0.6916086673736572, -2.458770275115967, -0.5028321743011475, 0.45875877141952515, 1.7037872076034546, -0.3201158046722412, 0.8848118782043457, 0.8411808013916016, 0.349799245595932, 0.20352879166603088, 0.44056689739227295, -1.473262906074524, 0.17192508280277252, 0.10091879963874817, 0.1441619098186493, -0.6956733465194702, -0.11654491722583771, -0.0703066810965538, 0.22171765565872192, 0.7423003315925598, -0.6493819952011108, -1.7428377866744995, -0.8084915280342102, -1.9414606094360352, -1.952831745147705, 0.02347957342863083, -0.0973430722951889, -0.43197840452194214, -0.8105667233467102, 0.4810619652271271, -0.06066704913973808, -2.7147216796875, 0.5831323862075806, 0.060397304594516754, 0.2116502970457077, 0.043692007660865784, 0.11819452792406082, -3.583616256713867, -0.15058206021785736, -0.1011064350605011, -0.10109731554985046, 0.43198466300964355, -1.4291859865188599, -0.03912436217069626, 0.8678735494613647, 0.006448825355619192, -1.0887800455093384, 1.433469533920288, -0.005483874119818211, -0.08656438440084457, 0.5588584542274475, -0.09997787326574326, -0.3844514787197113, 0.08551915735006332, 0.23869173228740692, 0.29739174246788025, 0.4867079555988312, 1.248740315437317, 0.4277034401893616, -1.543940782546997, -0.15173286199569702, -0.482945054769516, 0.3070567846298218, 1.0704866647720337, -0.7620287537574768, -2.0793042182922363, -0.7223227024078369, 0.2321174591779709, -0.5833758115768433, -0.24831824004650116, 2.1280999183654785, -0.41351085901260376, 0.1632044017314911, 0.31396806240081787, 0.8920692801475525, -0.002815738320350647, -0.5145546793937683, -0.1105191558599472, 2.665214776992798, -0.12101000547409058, 0.006434877868741751, -0.07917024940252304, -0.9482256770133972, -0.6419498920440674, 0.025668639689683914, 0.7065054178237915, 2.828902006149292, 0.5495483875274658, -0.09856245666742325, -0.25925731658935547, 0.6019273400306702, 0.28083789348602295, -0.18909473717212677, -0.5994632840156555, -0.44530802965164185, 1.2369396686553955, 0.16919107735157013, 0.1349870264530182, -0.24692374467849731, 3.4404196739196777, 0.2154332399368286, -1.0040318965911865, 0.15624576807022095, -0.05669880285859108, -1.300244927406311, 0.029507821425795555, -0.018418699502944946, 2.716484308242798, -1.246334195137024, 0.4056697189807892, 0.10757441818714142, 0.40200334787368774, 0.420490026473999, 0.17084528505802155, 2.561222553253174, 1.1234817504882812, -0.4924701452255249, -0.03070969134569168, -0.6907289624214172, -1.2430814504623413, 0.14108896255493164, -0.35531947016716003, -1.0730406045913696, 0.9326719641685486, 0.226453959941864, 0.9666427969932556, -1.1999949216842651, -0.07890846580266953, 0.5210641622543335, 0.04044569283723831, 0.4241848289966583, 0.3480154275894165, -0.7116573452949524, 0.4005778431892395, -0.1829008162021637, 0.31693682074546814, -0.34729307889938354, 0.7703162431716919, 0.033326465636491776, -0.9732835292816162, -0.5519466400146484, -1.0372451543807983, 0.0024886552710086107, 0.24775812029838562, 0.0933486595749855, 0.44947847723960876, 0.9398914575576782, 0.6040554046630859, -3.191801071166992, -0.29748740792274475, -0.32029393315315247, 0.07454366981983185, 0.24315311014652252, -0.3511736989021301, -0.1649322807788849, 0.0778796374797821, 3.9412412643432617, -1.016636610031128, -2.48363995552063, 0.9114524722099304, 0.25826355814933777, -2.199944019317627, 0.14862145483493805, 0.19097784161567688, -0.822300374507904, -0.8289219737052917, 1.946535587310791, 0.3214417099952698, -0.9783452749252319, -0.7978224158287048, 0.2765779495239258, 0.018707096576690674, 0.9958500266075134, 0.010372293181717396, 0.1558852344751358, -1.2663649320602417, 0.15265078842639923, 0.11448387056589127, 0.31278181076049805, -0.38910284638404846, 1.0073825120925903, -0.017212769016623497, 3.1567776203155518, -0.30632084608078003, -0.29874640703201294, -0.786449134349823, 1.2406107187271118, 0.09758542478084564, -0.005747492890805006, -2.4594414234161377, 0.3524863123893738, 0.13027134537696838, 1.5860923528671265, -0.5407527685165405, -0.6188558340072632, -0.19170908629894257, -1.047778844833374, 1.0911248922348022, -0.009175569750368595, 0.5506618618965149, -0.8429150581359863, -0.19039352238178253, -2.2392096519470215, 0.4901162087917328, -0.1236661896109581, 1.6464316844940186, 0.0436723455786705, -0.6187945008277893, -0.10172318667173386, -1.333593487739563, -1.7937054634094238, -0.025804445147514343, 1.4414619207382202, 0.4616312086582184, -0.17536978423595428, 0.04696636646986008, 0.2798512876033783, -1.385069489479065, -0.042142320424318314, -0.30956360697746277, -0.7521275281906128, -1.1065311431884766, -1.758147954940796, 0.17651121318340302, -4.650450229644775, -0.002072641160339117, 0.12722837924957275, -0.31249481439590454, 1.134766697883606, 1.8363659381866455, 0.16960063576698303, 0.13889172673225403, 0.8811387419700623, -0.3432646691799164, 0.3213415741920471, 0.01633024588227272, -0.23438045382499695, -0.6344722509384155, 0.06148487702012062, -0.1547306776046753, 1.4796602725982666, 0.22702272236347198, -0.6598122119903564, 2.023224353790283, -1.47980535030365, 0.1007349044084549, 0.8016512989997864, 0.10433176904916763, -1.4934613704681396, -0.18092137575149536, -0.5520525574684143, -0.12006448954343796, -0.8544394969940186, -0.15080615878105164, -0.09738317877054214, 0.4885796010494232, -0.08403419703245163, -1.266904354095459, -0.02482035756111145, -0.4172036349773407, 0.46025577187538147, 0.9169021248817444, -1.3722283840179443, -0.012605036608874798, 0.12484247982501984, 1.0558674335479736, 1.64598548412323, 0.24650292098522186, -0.17466934025287628, -0.9719526171684265, 0.29089826345443726, 0.18438716232776642, -0.5310012698173523, -0.0965694785118103, 0.34137335419654846, 0.028739701956510544, 0.3749956786632538, -0.17608603835105896, -0.011752315796911716, -0.8106226325035095, -0.01166900247335434, -2.3091161251068115, 0.21411682665348053, 4.015695095062256, 1.110471248626709, 0.4894503653049469, -0.4409369230270386, -0.6744688153266907, -0.306718647480011, -0.15991558134555817, 0.6995475888252258, -0.5825872421264648, 2.020409345626831, -0.9942782521247864, -0.048563096672296524, -0.2765563428401947, -0.04059401527047157, -0.2257143259048462, -1.8318489789962769, -0.14824031293392181, 0.19045527279376984, -0.10074841976165771, 0.5594947934150696, -1.378853440284729, 0.16750939190387726, -0.2460869997739792, 1.0405248403549194, -0.15549734234809875, 2.1740078926086426, -0.35378187894821167, -0.25621145963668823, -0.2582170069217682, -0.056333549320697784, 0.21780149638652802, -0.25346818566322327, 0.6664007306098938, 1.6637133359909058, -0.10268840938806534, -0.2375539094209671, -0.20055672526359558, 0.47251778841018677, -0.2524451017379761, -0.8756253123283386, -0.3091614842414856, -0.2358005940914154, 1.9607205390930176, 0.09352623671293259, 0.07684050500392914, -1.1848586797714233, 0.605767011642456, 1.455548644065857, -0.21152466535568237, 0.10415113717317581, 0.845991313457489, 0.09869179874658585, -1.2570358514785767, 1.8815484046936035, -0.012268751859664917, -1.6012505292892456, -0.10647473484277725, 0.022248094901442528, -0.08747217804193497, -0.18544885516166687, 2.2802395820617676, -0.25476038455963135, 1.5615811347961426, 2.152426242828369, 0.19813263416290283, -0.16744746267795563, -0.13664470613002777, 0.1582261621952057, 1.1253935098648071, -0.28068435192108154, -0.5910887718200684, 0.10543013364076614, 0.3031369745731354, 0.4073150157928467, 0.2428814172744751, -0.9969009757041931, -0.01627732813358307, 1.1450936794281006, 0.8791515827178955, 0.5618363618850708, -0.490919291973114, -0.33355316519737244, -0.15683259069919586, -0.6756735444068909, -0.053514931350946426, 0.29116132855415344, 2.095813751220703, 0.02029772661626339, 0.03666461631655693, -1.537302851676941, -0.11641176044940948, -0.2528306543827057, 0.1840425729751587, 0.039414357393980026, -1.1236226558685303, -0.4128377437591553, -0.05742965266108513, -0.1373550146818161, 0.773280918598175, 0.24594952166080475, 1.1471869945526123, -0.07002375274896622, -0.28517356514930725, -0.29407933354377747, 0.06272591650485992, -1.9116759300231934, -1.2555766105651855, 0.8469147682189941, -0.5356237292289734, 0.323009729385376, -0.46534621715545654, 0.3143618106842041, -0.5465494394302368, 0.6751488447189331, -0.11079191416501999, 0.2954238951206207, 0.06154133006930351, 0.02345244027674198, -0.15674303472042084, -0.033025480806827545, 0.7544565796852112, 1.4725662469863892, -0.6031371355056763, 1.0124938488006592, -0.9935779571533203, 0.14787566661834717, -0.4389924705028534, 0.7174755334854126, 0.3286239504814148, -0.415994793176651, -0.9046813249588013, 0.1314612179994583, -0.2403470128774643, -0.5973820090293884, -0.00020749303803313524, -0.17037257552146912, -1.3378915786743164, -1.3240485191345215, 0.8438363075256348, 0.49745291471481323, 0.37062475085258484, -0.0923321545124054, -0.1944168657064438, 1.8201521635055542, -0.6253873705863953, -0.17815950512886047, 0.13530932366847992, 0.04557815566658974, 0.012117153033614159, 0.04728357121348381, -0.5608752965927124, -0.07150717079639435, 0.4770297706127167, -0.6285658478736877, 1.0166691541671753, 2.110100269317627, 0.21840515732765198, 1.9460577964782715, 0.043488193303346634, -0.28241896629333496, -0.38286733627319336, -0.32262781262397766, 0.9757415056228638, 0.461394339799881, -0.22091728448867798, 1.7692008018493652, 0.9463610053062439, -0.4948177635669708, -0.3465151786804199, -0.2594861090183258, 0.5912875533103943, -0.9264904260635376, 0.01016371138393879, -4.159423828125, -0.32390350103378296, 0.5356550216674805, -0.022314399480819702, -0.823335587978363, -0.1368546187877655, -0.009838826954364777, -0.39744576811790466, 0.8121300935745239, -1.115127682685852, 0.6676573753356934, -1.4265283346176147, -2.61704421043396, 0.05193226411938667, 0.1558525562286377, 0.6141338348388672, -0.24510367214679718, 0.6304769515991211, -0.15171323716640472, 0.48544278740882874, -0.04934711381793022, 1.3059024810791016, 0.17669595777988434, 0.4428141117095947, -0.2506076395511627, -0.10885202139616013, -0.04412610083818436, -1.177651286125183, 0.10004287213087082, -0.6916606426239014, -0.649951159954071, -0.559526801109314, 1.051006555557251, -0.655284583568573, -0.09645745903253555, -0.1405486762523651, -0.28738296031951904, 0.20475612580776215, 0.1923215687274933, -0.6270710229873657, -0.026484031230211258, -0.4084048569202423, -0.06914831697940826, -0.5748597383499146, 1.2098571062088013, -0.13732129335403442, 0.1375475823879242, -1.1051517724990845, 0.5679897665977478, 2.060741901397705, -0.7035772800445557, 1.514789342880249, 0.17395921051502228, 0.47587746381759644, -0.09151751548051834, -0.03211400285363197, 0.11630990356206894, -0.08975017070770264, 2.983125925064087, -0.1315762996673584, -0.5517517924308777, 0.9834566116333008, 1.2873663902282715, -0.012536078691482544, -0.0874980092048645, 0.5872624516487122, 1.1260663270950317, -1.1005816459655762, -0.41601112484931946, 0.8725084662437439, -0.4904322624206543, 0.03864878788590431, -0.05241946503520012, 1.4271795749664307, 0.46594756841659546, -0.029599729925394058, -0.11524057388305664, 0.07697194069623947, -0.28427842259407043, 1.6639615297317505, 0.372157484292984, -0.8467644453048706, -0.06700330227613449, -0.19693920016288757, -0.2945772111415863, 0.33125245571136475, 0.1460617184638977, 0.18605004251003265, 0.0017500096000730991, 1.096954107284546, -0.8956514596939087, -0.06645288318395615, 0.6315394043922424, 0.34003376960754395, -0.708916187286377, 0.04686598852276802, 0.45556220412254333, 0.15186846256256104, 0.7229620814323425, -1.076088786125183, 1.506284475326538, -0.12128257751464844, -0.5370581746101379, 1.188845157623291, -0.3147106170654297, 0.2942262887954712, 0.37856563925743103, 0.2959515452384949, 0.1111387088894844, 0.5064675807952881, -0.042109064757823944, 0.14227859675884247, 0.08454679697751999, -2.269726037979126, -0.7730975151062012, 1.4569472074508667, -1.1160763502120972, 0.3015076220035553, -1.2542117834091187, -0.1989290863275528, 2.517361879348755, 0.27935799956321716, 2.803060531616211, 0.6249619126319885, 0.22678759694099426, -0.023065462708473206, 0.18660537898540497, 1.2762645483016968, 0.1772264689207077, 0.748746931552887, -0.016102004796266556, -0.5917049646377563, 0.0035299991723150015, 4.065798282623291, -0.44033703207969666, 0.013417243957519531, 4.467498779296875, 0.09463801234960556, 0.5744659304618835, 0.4786681532859802, 0.686288058757782, 0.05962597578763962, 0.8195545673370361, 1.0502452850341797, 1.0433937311172485, -1.6626651287078857, 0.17190270125865936, 0.19169756770133972, -0.13378749787807465, 0.34021177887916565, -0.18998020887374878, -0.15627023577690125, -0.22237491607666016, -0.08582649379968643, -0.1716107726097107, 1.2586724758148193, 0.8218222856521606, 0.026786332949995995, 0.18179777264595032, -0.4602462649345398, 0.24927031993865967, 0.7709774374961853, -0.030632751062512398, -0.6496011018753052, 2.129652976989746, 1.5959711074829102, 0.16718542575836182, -0.6688423752784729, 0.062169983983039856, -0.06011100858449936, 0.3492059111595154, -0.30700650811195374, 0.017218636348843575, 0.15636888146400452, -1.7019644975662231, -0.35540178418159485, 0.09079054743051529, -0.9025467038154602, 0.2759403884410858, 1.8031673431396484, 0.023602453991770744, -1.8371467590332031, -0.20024961233139038, -0.22970916330814362, 0.9327594041824341, -0.6178118586540222, 0.6406241655349731, 0.386386901140213, 0.10548065602779388, -0.1348300278186798, 0.18191078305244446, 0.9314552545547485, 1.5411988496780396, 0.14393065869808197, 0.033480748534202576, 0.467784583568573, -0.39288946986198425, 1.1584241390228271, -1.5208648443222046, 0.24371057748794556, -0.27951303124427795, -0.4735228419303894, -0.08622761070728302, -0.3348006308078766, 0.6445350646972656, -0.11305423080921173, -0.4719560742378235, -0.08945859968662262, -0.30047109723091125, 0.5779910087585449, -0.6103029847145081, -0.6248775124549866, 0.5611686110496521, -1.5859057903289795, -1.658473014831543, 0.030407199636101723, -0.5144752264022827, 0.7838548421859741, -0.21510735154151917, -0.03883588686585426, 1.167957067489624, 0.7429875731468201, 0.43492642045021057, -0.49200907349586487, -0.009432639926671982, 0.6016139388084412, -0.6906741261482239, -0.47587811946868896, 0.8155894875526428, -0.006521012634038925, 1.485853672027588, 3.420741558074951, -0.1252979338169098, -1.9512553215026855, 3.123774528503418, 0.3104924261569977, -0.7086830139160156, -0.07042030245065689, -0.9550817012786865, -0.3714750409126282, -0.08018006384372711, 0.07266727834939957, -0.03552306815981865, -2.8673269748687744, 0.022329121828079224, -0.43629080057144165, -0.6663196682929993, 0.5952231884002686, 0.05566408857703209, -1.6376545429229736, 0.17841607332229614, -0.09770558774471283, -0.8839179873466492, -2.3317418098449707, 0.5378164649009705, -0.9225460290908813, -0.05373572185635567, -3.2000532150268555, 0.011047177016735077, -0.4885596036911011, -0.1645115166902542, 2.499202013015747, -0.44611218571662903, 4.492885112762451, 0.3072206974029541, 0.04298898205161095, 0.0020652387756854296, 1.6250382661819458, 0.25507938861846924, 0.5557020902633667, 0.11915092915296555, 0.49731430411338806, 1.64211106300354, 0.23230712115764618, 0.30743589997291565, -0.011856251396238804, 0.013639163225889206, 0.09412601590156555, -0.7589563727378845, 0.7302541732788086, -0.06754714995622635, 1.0909538269042969, 0.7367777824401855, 1.1488515138626099, -0.015747297555208206, 0.06700854003429413, 1.997552752494812, -0.07374978810548782, 0.024604078382253647, 0.044414762407541275, 0.8866168856620789, 1.047026515007019, -0.3857010304927826, 0.795560896396637, -1.1978180408477783, 0.24332483112812042, 1.1924933195114136, -0.5229952335357666, 0.2077227681875229, 0.28933456540107727, 1.2740910053253174, -0.21022969484329224, 0.1760413646697998, 0.6581214666366577, 0.058960165828466415, 0.6726205348968506, -0.09193753451108932, 0.9380332231521606, 0.4009879529476166, 0.09448984265327454, 0.03145259618759155, -0.3402606248855591, 0.0027416949160397053, -0.15779727697372437, 0.8479307889938354, 0.2745092213153839, 1.016944169998169, 0.35238248109817505, 0.07757233083248138, -0.6587461233139038, 0.09488151222467422, 0.05978638678789139, -0.9318109154701233, 0.10831841081380844, 4.2814435958862305, 0.14222045242786407, -0.00014085836301092058, -0.33155131340026855, -0.18816113471984863, -0.5013263821601868, -0.491510808467865, -0.5792104005813599, 0.004857013467699289, -0.22903263568878174, 1.374006748199463, 0.6996749043464661, 0.4922143518924713, -1.2003885507583618, -2.074099063873291, 1.1036505699157715, 0.04440848156809807, 0.30339908599853516, 2.8892905712127686, 0.40926817059516907, 0.06722045689821243, 0.6289531588554382, -0.5818701386451721, 0.5138406753540039, 0.243267223238945, -0.4129219949245453, -1.6066709756851196, 0.006431455723941326, 1.331862211227417, -0.007719809655100107, 1.4915956258773804, 0.15317511558532715, -0.4626498520374298, 0.20765911042690277, 0.9321226477622986, 0.18694360554218292, -0.08477582782506943, 0.40080663561820984, -0.11786825209856033, -0.3456357717514038, 0.14564967155456543, -3.465054512023926, 0.09685565531253815, 0.5311441421508789, -0.6892539858818054, -0.38521111011505127, 1.9952514171600342, 0.41957852244377136, -0.008162004873156548, 0.9584916830062866, 0.24973131716251373, -0.019814398139715195, 3.0443806648254395, -0.46223264932632446, -0.0019480385817587376, 0.7869647145271301, 0.06257840991020203, 0.2217709869146347, -0.5266028046607971, 0.0718119814991951, -1.6843928098678589, 1.1500859260559082, -0.40500718355178833, -0.1796179711818695, -0.5361328721046448, 0.03752857446670532, 1.2395566701889038, 1.2152889966964722, 0.6859528422355652, 0.042604960501194, -0.09429318457841873, -0.05840439349412918, -4.741398811340332, -0.23872055113315582, 0.4485029876232147, -0.018602905794978142, 0.6362280249595642, 1.6758943796157837, 0.24188587069511414, -0.6336716413497925, -1.5619179010391235, -0.18783657252788544, 0.2545260190963745, 1.7051482200622559, 0.08354480564594269, -0.24197937548160553, -0.6654439568519592, -0.5131071209907532, 1.2774220705032349, 0.02082025073468685, -0.4901967942714691, -0.5784628391265869, -0.2455853819847107, -1.5224462747573853, -0.1004001647233963, -0.23940153419971466, -0.08255652338266373, -0.03176940232515335, 2.907076835632324, -0.003045310266315937, -0.18707913160324097, 0.40178418159484863, 0.557563304901123, -0.24333079159259796, -0.21956542134284973, 0.029646458104252815, -0.4458475708961487, -0.811699628829956, 0.20761576294898987, -0.966792106628418, 3.1848461627960205, -0.9326657652854919, -0.8467022776603699, -0.47388502955436707, 0.021741872653365135, -2.2253684997558594, 0.4028708040714264, 0.08091707527637482, 0.6940233707427979, 0.03428811952471733, -0.003964254632592201, 0.09362487494945526, -0.21378856897354126, 1.0136715173721313, 0.6790373921394348, 0.623685359954834, -0.6900067925453186, -0.11815573275089264, -0.11150617152452469, -0.06147635355591774, -1.5933274030685425, -0.33212170004844666, 1.773435115814209, 0.017043914645910263, 0.2922757863998413, 0.00836754310876131, 0.0074377646669745445, -0.00464773690328002, 0.07064411044120789, -0.4737338721752167, 0.353625625371933, 0.02490193024277687, 2.103825807571411, 0.10172995924949646, -1.665649652481079, -0.07287096232175827, 2.1667375564575195, -2.4758918285369873, 0.396365225315094, -3.443882465362549, -0.0045555452816188335, -0.12434424459934235, 0.949244499206543, 0.10368651151657104, 0.07082091271877289, -0.6453704833984375, -0.08912083506584167, 0.039408762007951736, -0.3323829770088196, -0.18220151960849762, 0.32361578941345215, 0.5628358125686646, 0.02240324392914772, -0.06634446233510971, -0.6181154847145081, 0.3358146846294403, 0.8432677984237671, 2.286717176437378, -0.19140800833702087, 0.8289602994918823, -2.500368595123291, 1.3964385986328125, 0.3646373152732849, 0.6209482550621033, -0.028713952749967575, -1.3069453239440918, -0.09930283576250076, 1.5600048303604126, 0.831089973449707, 0.51837557554245, 0.29965198040008545, -0.352418452501297, 0.6317518949508667, 0.22896353900432587, 1.5565998554229736, -0.14410610496997833, -0.40883469581604004, 0.3752613067626953, -1.8815336227416992, 0.10604199767112732, 1.080869436264038, 0.7620832324028015, 0.09277801960706711, 0.43662387132644653, -0.13863079249858856, 0.24652057886123657, 0.3106424808502197, -0.2492167055606842, 0.05930543318390846, -0.09634457528591156, -0.943717896938324, 0.19008706510066986, -0.3358953893184662, 0.14678221940994263, -0.034237589687108994, -0.11838079988956451, 0.9756239652633667, -0.31412622332572937, 0.5335452556610107, 1.5658175945281982, -0.1919097900390625, -0.8306061625480652, 1.5061626434326172, -0.22870822250843048, -0.08317472040653229, 0.02857603318989277, 1.1726051568984985, -0.006370714865624905, -0.27655521035194397, -0.201973557472229, -0.10157693177461624, -0.14998076856136322, 1.6956883668899536, 0.3572114408016205, 0.3724100589752197, -1.8198062181472778, -0.20490942895412445, -0.017584078013896942, 0.06817281246185303, 0.48962974548339844, -0.7283045649528503, -0.9499653577804565, 0.5437283515930176, -0.4987429082393646, -0.06713701039552689, -0.7047123312950134, -0.610062837600708, -0.30094030499458313, -0.3552819490432739, 0.34590402245521545, -0.30736085772514343, 0.02218404784798622, 0.00686604855582118, 0.10505176335573196, -1.1754491329193115, 0.1445188671350479, -0.4129050672054291, -0.3256465494632721, 0.6454792022705078, 0.023686828091740608, -0.32467904686927795, 2.072932243347168, 0.3063158392906189, -0.06584925949573517, -0.450832337141037, -1.8152878284454346, 0.04695269837975502, 0.5259458422660828, -0.5368784666061401, 0.3835190534591675, 1.1496793031692505, -0.17325358092784882, -0.3911367952823639, 0.06963599473237991, 0.42930683493614197, 0.1201002299785614, 1.7664915323257446, 0.7888568639755249, 0.004259842447936535, 1.6948025226593018, 0.994755208492279, -0.16081173717975616, 1.1715489625930786, -0.04826904460787773, -1.2813369035720825, 0.3518463671207428, -0.1354144662618637, 0.24363024532794952, -1.2974532842636108, -0.06488971412181854, -0.5149760842323303, 2.5721304416656494, 0.5236641764640808, 0.23215192556381226, -0.2621998190879822, 0.5455887913703918, 1.303658127784729, -0.794904351234436, 0.02105364389717579, 0.21025334298610687, 1.3446056842803955, 0.33022263646125793, -0.13130898773670197, 0.07258011400699615, -0.4332699179649353, 0.6462544202804565, -0.18419434130191803, -1.4297178983688354, 0.0581992007791996, 0.7617337703704834, -0.03471631184220314, 0.0696190893650055, -0.0867101177573204, 0.46467527747154236, 0.05845861881971359, 0.22627171874046326, -0.0443372018635273, 0.21093320846557617, -0.271441787481308, 0.020165594294667244, 0.7337768077850342, -0.3617970645427704, 0.7433868646621704, 4.201419830322266, -0.04056014120578766, -0.21155484020709991, 0.3625505566596985, -0.05977707356214523, 0.2781437933444977, 0.20726796984672546, -1.1771663427352905, 1.0347704887390137, 0.5607370138168335, 0.028468113392591476, 0.3050402104854584, 0.08469084650278091, 0.15659822523593903, -0.03985895588994026, -0.010990089736878872, 2.126190662384033, 0.642177402973175, 2.2017576694488525, -1.4116476774215698, 0.3638089895248413, -0.19944879412651062, 0.3977993130683899, 0.1609765887260437, 0.1718566119670868, -0.09843504428863525, -0.4211747944355011, -0.4227929711341858, -1.5030936002731323, -0.4297402799129486, -0.287725031375885, 1.022268533706665, 0.0975046306848526, 0.45073187351226807, 0.14302925765514374, -0.771192193031311, 0.1711847186088562, 0.055177558213472366, -1.3772552013397217, 2.7941744327545166, 2.366760015487671, 0.04984395205974579, 0.6636064052581787, 0.7982286810874939, 0.08924566954374313, 1.3818796873092651, -1.1225699186325073, 0.4703476130962372, 1.1548818349838257, -0.32396334409713745, 0.23700283467769623, 0.04443754255771637, 0.4051169157028198, -0.20434528589248657, 0.3736451268196106, 1.053152084350586, 0.26537710428237915, 0.33476096391677856, 0.1987069696187973, -0.002062029205262661, -1.7120604515075684, 0.31904366612434387, -0.8387983441352844, 0.8340927958488464, 0.5171443819999695, 0.5735167860984802, -0.19240300357341766, -0.0035602727439254522, 0.1287970244884491, 0.38272807002067566, 0.5881860852241516, -1.125333547592163, 1.49907386302948, 0.22125907242298126, -0.227415531873703, -0.4057789444923401, -1.0958212614059448, 0.2751862108707428, 3.1731765270233154, 0.10053300857543945, -0.09247254580259323, -0.6392437815666199, -1.6681510210037231, -0.1417243480682373, -1.8982259035110474, 0.6417869925498962, 0.19502392411231995, -0.029978256672620773, 0.14673559367656708, 1.564156174659729, -0.19515137374401093, -0.7764649391174316, -0.04932749271392822, 1.386201024055481, 0.06758872419595718, 0.8933510184288025, 0.49445709586143494, 0.42002877593040466, 1.0330034494400024, 1.3087555170059204, -0.004588941112160683, 2.860915422439575, 9.526428584649693e-06, 1.3967020511627197, 0.0013603126863017678, 0.6231579184532166, 2.0533320903778076, 0.798988401889801, 0.6902644038200378, 0.22322914004325867, -0.8291507959365845, -0.4763210117816925, 1.6364320516586304, 0.7115804553031921, 0.46480420231819153, 0.08357792347669601, 0.167560413479805, -0.011533095501363277, 0.9603602886199951, 0.41840603947639465, 0.01437593623995781, 3.7570817470550537, -1.610413908958435, 4.843364715576172, 0.3915466368198395, -0.2691497206687927, 1.182884931564331, -0.612213134765625, -0.33766499161720276, -0.8290138244628906, -0.1278715580701828, -0.3819260597229004, 0.4832414984703064, 0.3870965838432312, -0.2920096218585968, -0.2769477665424347, 0.678818941116333, -0.0002892606717068702, -0.14405639469623566, 0.050782348960638046, -0.09075503796339035, 1.5266141891479492, 0.011245123110711575, -0.42848995327949524, 0.2062578946352005, -0.8033515810966492, 0.4453543722629547, -1.378855586051941, 0.7316663861274719, -0.11297421157360077, -0.006304153706878424, 0.512890636920929, -0.7224289774894714, -0.21905897557735443, 0.00016985931142698973, -0.7763298749923706, -0.13204379379749298, 0.13453039526939392, -0.23202699422836304, 0.11261823773384094, 0.104156993329525, 0.04506939277052879, -0.5119474530220032, 2.8573405742645264, 0.09962540119886398, -0.089596688747406, 0.29942190647125244, 0.5452682971954346, -0.2519862949848175, -1.5567885637283325, 0.24122045934200287, 0.2491658478975296, 3.2299041748046875, -1.6540429592132568, 0.0066506206057965755, 0.5757169127464294, 2.0570695400238037, -0.36815544962882996, 0.005646436009556055, 0.5613613724708557, 0.5552337169647217, -0.5353547930717468, 0.0019358523422852159, -0.12068622559309006, 0.2954581677913666, 0.3858281373977661, -0.7944334149360657, -0.675447940826416, 0.1409209668636322, 2.2911760807037354, -0.16530989110469818, 0.08492856472730637, -1.436130166053772, -0.2984931170940399, -0.48475024104118347, -0.09955272078514099, -0.31712859869003296, 2.6190924644470215, 0.890451192855835, 0.11312045902013779, 0.22419531643390656, -0.1623346358537674, 0.15537679195404053, -0.6410788893699646, 0.06734580546617508, 0.16828253865242004, 0.005393628962337971, 0.07658570259809494, -2.286511182785034, -0.5374683737754822, 0.10924189537763596, -0.002935107098892331, 1.0486773252487183, 0.10298895835876465, 2.1632144451141357, 0.00970050785690546, -0.09517588466405869, -0.6001659631729126, 0.14421899616718292, 0.16678573191165924, 0.182171031832695, -0.10292384773492813, 0.9810189604759216, -1.4305546283721924, -0.41356360912323, 1.9296519756317139, 0.4995552897453308, -0.35815122723579407, -0.8443630337715149, -2.0374624729156494, 0.9724175333976746, -0.842576265335083, 0.34268832206726074, 0.007228806149214506, -1.5426398515701294, 0.4395263195037842, -0.12088483572006226, -0.8843650817871094, 0.5475488305091858, -0.28530293703079224, -0.0036294132005423307, 0.2386094182729721, 0.2113521546125412, -0.23814786970615387, 0.12917029857635498, 2.913691282272339, -1.4770900011062622, -0.15831154584884644, 0.4227507710456848, -0.09331496059894562, 0.7535040974617004, -0.14782175421714783, -0.2392093688249588, 1.1615161895751953, 0.4051680862903595, 0.23005394637584686, 0.47604137659072876, -0.35717710852622986, 0.013385530561208725, 0.9315602779388428, -0.1158783957362175, -0.09191150963306427, -3.0998573303222656, 0.3634978234767914, -0.2735511064529419, 0.9541698694229126, 2.016756296157837, -0.01234038919210434, -0.31330499053001404, -0.38725629448890686, -0.01407962292432785, 0.8780134916305542, -0.2485101968050003, -0.5792408585548401, -1.2522531747817993, 0.2605092525482178, -0.2795082926750183, -0.8535876870155334, -0.03665366768836975, 1.1869618892669678, 0.00999381672590971, -0.35877105593681335, -0.009135492146015167, -0.022713355720043182, -0.5793129801750183, 0.34232598543167114, -0.32060298323631287, 0.044853486120700836, -0.852236270904541, 1.7628486156463623, 1.0144872665405273, -0.05832619220018387, 0.8900805711746216, 0.023487448692321777, -0.6805784702301025, 0.013859380967915058, 0.06894609332084656, -1.1068224906921387, 0.24255511164665222, 0.1952330470085144, -1.4414191246032715, 0.5383201837539673, 0.017895961180329323, -1.3346681594848633, 0.39867672324180603, -0.6811190247535706, 0.9352222084999084, -0.7576841115951538, -1.4260838031768799, 0.4853491187095642, -0.2763054370880127, 1.727268934249878, -0.7077111601829529, -0.06848058849573135, -1.03353750705719, 0.36493757367134094, 0.5525860786437988, -0.4525333642959595, 0.006842785980552435, 0.34814518690109253, -1.628836989402771, 0.35987159609794617, 0.004748399835079908, 1.707002878189087, -0.017206799238920212, -0.038636256009340286, 0.7297710180282593, 1.0233776569366455, -0.029565293341875076, -0.07634206116199493, 0.14311718940734863, 0.33240506052970886, -0.23644918203353882, 0.3174339830875397, 0.756410539150238, -0.19250790774822235, -0.15189941227436066, -1.4441198110580444, -0.8584911823272705, -0.727783739566803, 1.5572514533996582, 0.8522180318832397, -0.15446026623249054, -0.6602229475975037, 0.9058650732040405, -0.12329230457544327, -0.034532856196165085, 0.24472908675670624, -0.012645612470805645, 0.36365941166877747, 0.9790236949920654, 0.0010374148841947317, 1.9374626874923706, -0.6506320834159851, -0.3916124105453491, 0.5774701833724976, -0.4297696650028229, 0.10836543887853622, -0.13695144653320312, -0.928683876991272, -0.006395050324499607, -0.7134045362472534, 0.7796733975410461, 0.0461217537522316, 0.11239328235387802, 1.085525631904602, 0.6840564012527466, 0.21175125241279602, -0.15470261871814728, 0.21152013540267944, -0.1928352564573288, 0.33593109250068665, 0.1212117150425911, 0.4764738082885742, -1.2340810298919678, 0.05364366993308067, 0.20464904606342316, -0.0672868937253952, 0.15417815744876862, 0.18942499160766602, -0.06807320564985275, 0.05563448369503021, 0.014287719503045082, -0.37025415897369385, -0.7416669130325317, 1.0577778816223145, -0.43746817111968994, 4.083448886871338, 0.6379851698875427, -1.3104270696640015, 2.551429271697998, 0.3170681297779083, 0.03746134787797928, 0.07239292562007904, 0.2760045826435089, 0.12164106965065002, -0.1851835697889328, -0.3316429853439331, -0.12303058803081512, -1.1050249338150024, -0.3952861428260803, -0.7721704840660095, 0.1207246258854866, -0.39140865206718445, -0.14428143203258514, -0.009999374859035015, 0.5875431299209595, -0.008968770503997803, -0.39625951647758484, -0.2449304610490799, -0.007965787313878536, -0.24311423301696777, 0.4591062068939209, -0.10903458297252655, 0.5238280892372131, -0.007947955280542374, -0.5017198920249939, -0.3033474087715149, -0.610447347164154, -0.07710552215576172, 2.279120683670044, -0.006050890311598778, 0.061042387038469315, 0.06643252074718475, -0.25126299262046814, -0.5073632001876831, 0.7681846022605896, -1.0288848876953125, 0.3712252378463745, -0.04477853327989578, 0.38044285774230957, -0.09442394971847534, 0.4183989465236664, -0.5667884945869446, 0.47757643461227417, -0.5275556445121765, -1.5282180309295654, 0.28118404746055603, -0.19781073927879333, -0.03524252399802208, 1.4056270122528076, -0.8598840236663818, 0.42092838883399963, -0.43797585368156433, -0.15224522352218628, -1.2648199796676636, 0.008289577439427376, 0.8661274909973145, 0.05986645817756653, 0.6501854658126831, -0.3887762725353241, -1.454057216644287, 0.16338296234607697, 0.03431589528918266, -0.4302230477333069, -0.0070383246056735516, 0.10566788911819458, 1.1351407766342163, 0.22672031819820404, -0.1944582462310791, -3.812981367111206, -0.19659726321697235, -0.5644896626472473, 0.5249086618423462, 2.751067876815796, -0.08735007047653198, -0.24880851805210114, 0.3427152633666992, -0.4953062832355499, -0.13764823973178864, 0.043230343610048294, 1.522186040878296, 3.0209741592407227, -0.2162162959575653, -0.21467533707618713, 1.1646382808685303, -1.0449804067611694, -0.01572277769446373, 0.024684347212314606, -0.4216957986354828, 6.008416652679443, 0.539419412612915, 0.06067638844251633, 0.17404931783676147, 3.4072675704956055, -1.8232197761535645, 1.117987036705017, -0.1735447347164154, -1.3995308876037598, -0.6342073678970337, -0.5741076469421387, 0.8440604209899902, -0.07092321664094925, -1.0275009870529175, 0.2568541467189789, -0.12599483132362366, -0.05863013491034508, -0.27875298261642456, -0.2655213177204132, -1.889230489730835, -1.7363454103469849, 0.07092002034187317, 0.15608316659927368, 0.02746465988457203, 0.1756628304719925, 2.2001969814300537, 0.27350306510925293, 0.015265780501067638, 2.2611703872680664, -0.3312276303768158, 1.5484999418258667, 0.2767793834209442, 0.5941219925880432, -0.05148524045944214, -0.6006894707679749, 0.13554181158542633, 0.7086563110351562, 0.05751059204339981, 0.0027131212409585714, 0.00010842222400242463, -0.13838501274585724, 2.7825570106506348, -0.5666998028755188, -1.2984907627105713, 2.0672876834869385, -0.710477888584137, -0.8254598379135132, -0.41155946254730225, -1.3968762159347534, 4.775707721710205, 0.04403762146830559, 0.2714706361293793, -0.41864725947380066, -0.19870741665363312, -0.5012504458427429, -0.09784982353448868, 0.007236855570226908, -0.055479105561971664, 0.37751829624176025, 0.19916276633739471, -0.9767009019851685, 0.2514565587043762, 0.010436128824949265, 0.13848520815372467, -0.14460361003875732, 0.05611928924918175, 1.8367211818695068, 0.503813624382019, -2.1527421474456787, -0.0586409717798233, -0.21878375113010406, -0.04351634532213211, -0.5528466105461121, -0.7006826996803284, 0.607081949710846, 0.01668110117316246, -0.520771324634552, 0.41917598247528076, -0.020526109263300896, -0.05939486622810364, 1.068169355392456, 0.053429994732141495, 1.1224116086959839, 1.0241177082061768, -2.4623987674713135, 0.1180776059627533, -1.7882006168365479, -1.391786813735962, 0.11023182421922684, 0.07862673699855804, 0.15321104228496552, -0.06893604248762131, 0.2473059445619583, 0.2764791250228882, 0.9982629418373108, -0.0044616954401135445, 0.4523124396800995, -1.5795718431472778, 0.40277084708213806, 0.24101094901561737, 0.09416545182466507, -0.34837716817855835, -2.381232261657715, -1.8917328119277954, -1.2261923551559448, -0.0593138113617897, 0.3197925388813019, 0.4438394606113434, -0.7268511652946472, -0.4679850935935974, -0.2870968282222748, -0.8744503855705261, -0.712984561920166, -1.0587184429168701, -0.023253198713064194, -1.7493019104003906, -0.0019073232542723417, 0.13844622671604156, 0.0511162243783474, 1.5131150484085083, 0.04727915674448013, 0.2072964459657669, 0.5189982056617737, -0.1238207295536995, -0.13048139214515686, -1.8212084770202637, -1.3916057348251343, 0.6827359199523926, 1.1637684106826782, 0.09532416611909866, -0.6896682977676392, 0.39193159341812134, -0.1799442321062088, -0.1475144326686859, 0.150967538356781, -0.14881768822669983, -0.18773147463798523, 3.277801036834717, -0.36791589856147766, -0.003737807972356677, -0.1582093983888626, -0.8076791167259216, 1.9269459247589111, 0.10639256983995438, 0.17404095828533173, 0.032062940299510956, -0.428112268447876, 0.23744048178195953, 0.2009744644165039, 1.0961663722991943, 1.7080068588256836, -0.0028178931679576635, 0.3957095444202423, 1.2533819675445557, 0.31035691499710083, -0.011967673897743225, -0.3982544243335724, 0.36736705899238586, 0.12053759396076202, -3.219331979751587, -0.01490087527781725, -0.1772681474685669, 0.015036141499876976, -0.3404398262500763, -0.06744137406349182, -0.10382673144340515, -0.8386001586914062, -0.06297770887613297, -0.974989116191864, 1.4309486150741577, 0.2518371641635895, -0.14525140821933746, -1.4331074953079224, -3.584155797958374, -0.0045995935797691345, -0.39173799753189087, 0.0075152269564569, -0.347914457321167, -0.1782098114490509, -1.04216730594635, -0.4894884526729584, -0.08399303257465363, -0.03810455650091171, -0.06965811550617218, -1.3381551504135132, 0.06746000051498413, -1.2719866037368774, 0.30743643641471863, -0.6502392292022705, -0.24579188227653503, -0.4786085784435272, 2.2207400798797607, 0.3460640609264374, 0.29007795453071594, -0.3774668872356415, -1.0708634853363037, -1.198901653289795, -0.482788622379303, 1.7956827878952026, 0.17345207929611206, 0.3395931124687195, -0.23985867202281952, 0.17755219340324402, 0.4228232800960541, -0.9364312887191772, 0.9611976146697998, -0.29898086190223694, 1.8898403644561768, 0.38973864912986755, -0.586367130279541, 2.483306407928467, 1.3256890773773193, 1.013449788093567, -1.5129539966583252, -0.6796377301216125, 1.8844510316848755, 0.8175581097602844, -0.06004471704363823, 0.09319154173135757, 0.30342069268226624, 0.2747633755207062, -0.005308405961841345, 0.3889125883579254, 0.11793959885835648, 0.023272674530744553, 0.09684479981660843, -0.3003377914428711, 0.8316178321838379, 0.859558641910553, -0.04754922166466713, -0.9098305106163025, 0.24920609593391418, 0.0037321047857403755, -0.3325577974319458, 0.5578964352607727, -0.4172269403934479, -0.347602903842926, -0.08194184303283691, 1.0522555112838745, -0.6769154667854309, -0.23386654257774353, -0.004823941737413406, -0.3159245252609253, 1.384476661682129, 0.07241039723157883, 0.6209686994552612, 0.14119413495063782, 0.43002671003341675, -0.061175134032964706, -3.0557398796081543, -0.016077904030680656, -0.3115861713886261, 1.1432905197143555, -0.00450656283646822, -1.8005306720733643, -0.02224939502775669, -0.11851869523525238, 0.49699532985687256, -0.7725638747215271, 0.13602490723133087, -0.386798620223999, 0.5488989949226379, 0.388355553150177, 0.2643131911754608, -0.21345610916614532, 1.1255046129226685, -0.7389854788780212, -0.8815085887908936, 0.2946890890598297, -0.42715439200401306, -0.9808382391929626, 0.050552889704704285, 0.16792789101600647, 1.206182599067688, 3.7044332027435303, -0.04041149094700813, -0.5783069133758545, -0.9203939437866211, 2.8827574253082275, -0.06885368376970291, -0.14569175243377686, 0.07210180163383484, 0.043005093932151794, -1.2269630432128906, -0.008874576538801193, -0.5638297200202942, 0.00946128461509943, -0.9297186732292175, 0.04640090465545654, 0.04826074093580246, -0.23062410950660706, 1.731469988822937, -1.6753082275390625, 1.0347610712051392, -1.5784399509429932, -1.9531508684158325, -0.13214191794395447, 0.14644505083560944, 1.2092748880386353, -1.0463992357254028, -0.6343744993209839, 1.242640495300293, 0.19838136434555054, -0.1759897768497467, 1.1208488941192627, 1.6526323556900024, -0.585604727268219, 0.4855051040649414, -1.1366468667984009, -1.0860495567321777, -0.16961553692817688, -0.13070577383041382, -0.2566336691379547, -0.012745408341288567, 0.11492549628019333, -0.6045968532562256, 1.12647545337677, 0.5858650207519531, 0.05161692202091217, -2.027780055999756, 1.2494945526123047, 0.6827834248542786, -0.08084069937467575, 0.027783073484897614, -0.33958253264427185, -0.36945024132728577, -0.34989991784095764, -0.20871342718601227, -0.2894552946090698, -0.5802898406982422, 0.07434744387865067, 0.2725968360900879, -3.721022844314575, 0.1631617248058319, -1.1124054193496704, 0.06070045381784439, -0.02448255941271782, 0.2232188582420349, 0.18380147218704224, -0.5988421440124512, -0.39637312293052673, -0.21212147176265717, 1.7600079774856567, -2.841872215270996, 0.3870103061199188, -0.972352921962738, 1.3850557804107666, 0.0870421975851059, -0.6431308388710022, -0.2567864954471588, -1.2096673250198364, 0.050715044140815735, 3.605483293533325, -0.008666747249662876, 0.7014480233192444, -0.07091659307479858, -0.008541093207895756, 0.7820072174072266, 0.22183986008167267, -0.014697282575070858, -0.8028927445411682, 0.7305861115455627, -0.13039909303188324, -0.05788016691803932, -0.2061726152896881, -0.2312915176153183, -0.014697142876684666, -0.39819902181625366, -0.033828482031822205, 2.087392807006836, 1.9868555068969727, -0.03213542327284813, 0.8297170996665955, 1.484086513519287, -0.03730743005871773, -4.132949352264404, 0.13659943640232086, 0.23976950347423553, -0.11719991266727448, 1.2648934125900269, -0.9289255142211914, 0.030408939346671104, -0.008473598398268223, -1.129839301109314, 0.5149285793304443, -0.00044303692993707955, 0.4857880771160126, -1.2942535877227783, -0.1177503913640976, -0.7450674772262573, 1.724434733390808, -0.01373106800019741, 0.047065507620573044, 0.2579857409000397, 0.6947458982467651, 0.04621027410030365, 0.4692084789276123, 0.08358388394117355, 0.12052164226770401, -0.36784628033638, 0.33661141991615295, -0.8121451735496521, 0.10788709670305252, 0.10235170274972916, 0.9549607038497925, -0.3583488464355469, -0.15054136514663696, -0.961414635181427, 0.3892230987548828, -0.4417084753513336, -0.3255392611026764, -0.0652681514620781, 2.041478157043457, 0.05510612204670906, 0.2995101511478424, 0.1459856629371643, -0.49926528334617615, -0.13696768879890442, 0.7179843187332153, 0.8851633071899414, 0.13825048506259918, -1.0825507640838623, -0.4177902340888977, 0.8004144430160522, 1.0357168912887573, -0.6693692803382874, 0.06073762848973274, 0.06805504113435745, 0.34347355365753174, 0.3956605792045593, 0.3471881151199341, -2.2519192695617676, 0.11143410950899124, -1.0606344938278198, 0.32728156447410583, -0.5229555368423462, -0.5890650153160095, -0.21409863233566284, -0.0007954192697070539, -0.08652333170175552, -1.1316930055618286, -0.10489252209663391, -0.42854368686676025, -0.7691529393196106, 0.5186020731925964, -0.9978833794593811, -0.17164325714111328, 0.6019072532653809, 0.11309347301721573, 0.015446055680513382, 2.27103853225708, -0.272584468126297, -0.20907072722911835, -0.21553654968738556, -0.5384852290153503, 0.337674081325531, 0.03429931774735451, -1.1628650426864624, -0.9347166419029236, 0.1547318547964096, 0.05590088292956352, 2.850440502166748, 0.13165625929832458, -0.0012123852502554655, 0.506488561630249, -0.15120768547058105, 0.5270587801933289, 0.5372249484062195, -0.09397843480110168, 0.4634724259376526, 0.021018894389271736, 0.1825895756483078, 1.0752975940704346, 0.5957111716270447, -1.213675618171692, -0.4868313670158386, -2.3199622631073, 0.002480195602402091, -0.8468286991119385, -1.892662525177002, -0.0031544568482786417, 0.07503195106983185, -0.14829036593437195, 0.5984524488449097, 3.9584860801696777, 0.520776629447937, -0.03805302828550339, -0.017341241240501404, -0.19621486961841583, -1.1114370822906494, -0.6004283428192139, -1.846268892288208, 0.18382611870765686, 0.7371607422828674, -0.5626420378684998, -0.022018540650606155, -0.00792649108916521, -0.47106191515922546, 0.3064005970954895, -0.006568597164005041, 0.8809840679168701, 0.13319985568523407, 0.4278630018234253, -0.02116914838552475, -0.15170033276081085, -0.11279305070638657, -0.16531381011009216, -1.5007578134536743, -0.02756849303841591, -0.015879221260547638, -0.2000313401222229, -0.04357930272817612, -1.1859347820281982, 0.8578683137893677, 1.1312931776046753, 1.6132447719573975, -0.4080859124660492, -0.40423280000686646, 0.7590230107307434, -0.8183863759040833, -0.5677714943885803, -0.5404067039489746, 0.5519919991493225, 0.6090036630630493, 0.11518959701061249, -0.059382081031799316, -1.1669211387634277, 0.031096508726477623, 1.0670554637908936, -0.2986116409301758, -0.029201822355389595, 0.06633239984512329, 0.533684253692627, 0.2358083575963974, -0.29266050457954407, 2.485471725463867, 0.5860358476638794, -0.21097944676876068, -1.044772982597351, 0.19092382490634918, -0.31418928503990173, -0.01343494188040495, 0.10113970935344696, 0.024558762088418007, 0.6524815559387207, -1.2454993724822998, 0.032052647322416306, 0.024536242708563805, -2.7242934703826904, -0.09389778971672058, -0.1703304946422577, -0.1903609335422516, -0.2820349931716919, 0.9206070899963379, -0.3464913070201874, -3.656999111175537, -0.6212413907051086, 0.03520407900214195, -0.7989853024482727, 1.8241887092590332, 0.34639519453048706, 0.7825239896774292, 0.8095642328262329, -0.7923865914344788, -0.02557867020368576, 1.7161377668380737, -0.6992694735527039, 1.9821146726608276, 0.2475607842206955, 0.27764102816581726, -1.001433253288269, -0.6258326768875122, 0.2017308622598648, 1.210995078086853, 0.9321097135543823, 0.004592141602188349, -0.6976262927055359, 1.217755675315857, -0.07607623934745789, 0.0008240087190642953, 0.48038649559020996, 0.0942416787147522, -0.36405012011528015, -0.07518546283245087, 0.6647054553031921, 0.23702944815158844, -0.655616819858551, 0.29256725311279297, -0.6453308463096619, 1.7019389867782593, -0.26723137497901917, 2.6966943740844727, 0.01267299149185419, -0.19760975241661072, 0.006083289161324501, 0.6346325874328613, 0.16745270788669586, -1.794853687286377, -0.5530165433883667, 2.142362356185913, 0.5481874942779541, 0.14595440030097961, -0.41062456369400024, -0.08310937136411667, 0.8787907361984253, -0.4842003583908081, 0.1740940660238266, -0.7634947299957275, -3.1057915687561035, 0.061799194663763046, -0.71574467420578, -0.023497406393289566, 0.048942893743515015, -5.872338771820068, 0.3735952377319336, 3.191025733947754, 0.1656855046749115, 0.24312430620193481, 1.7486119270324707, -0.647794783115387, -1.71144700050354, 0.47416260838508606, 0.07857082784175873, -0.07157513499259949, 1.3823195695877075, 0.39056745171546936, -0.14032770693302155, -0.399247407913208, 0.232381671667099, 0.24401560425758362, 0.14835073053836823, -0.22349287569522858, 1.6157740354537964, 0.20168064534664154, 0.01668708398938179, 1.4804869890213013, -0.8595672249794006, 0.12215239554643631, -0.13161630928516388, 0.01953568495810032, 3.1461689472198486, -0.046311456710100174, -0.005601349752396345, -0.9577625393867493, -0.4544043242931366, -0.13846087455749512, 0.0027989884838461876, -0.19759489595890045, 0.19946648180484772, -0.35974249243736267, 0.16066671907901764, -0.2387303113937378, -1.067093849182129, 0.05310365557670593, 0.7155309319496155, -1.183700442314148, 0.08201488107442856, 2.0480940341949463, 0.07949285954236984, 1.0772043466567993, -0.08665050566196442, -0.09522045403718948, 0.06172284111380577, 0.019318686798214912, -0.0796072855591774, -0.5135120153427124, 0.1022016629576683, -0.11714335530996323, -0.1316952258348465, 0.5237811207771301, 1.185223937034607, -0.707012951374054, 0.6338357329368591, 1.0836727619171143, -1.1037687063217163, 0.48084893822669983, 0.7014071941375732, -2.5073962211608887, 0.004286447074264288, -0.06501442939043045, 0.3872133791446686, 1.4168752431869507, 0.6962379813194275, 0.8970423340797424, -0.011619996279478073, 0.40895968675613403, 0.8521052002906799, -0.39584824442863464, 0.005627304315567017, 0.5951753258705139, -0.038797687739133835, -0.0439746156334877, 0.4884222447872162, -0.029967844486236572, -1.0136139392852783, -0.11658678203821182, 0.18510355055332184, 0.1987205296754837, -0.2731834650039673, -0.8065004944801331, 0.7451477646827698, -0.572407603263855, -0.8783885836601257, -0.07288627326488495, -0.10877599567174911, 1.3126935958862305, 0.05028938129544258, -0.07024259865283966, 0.8819653391838074, -0.703119695186615, 0.5360602736473083, -0.6317170858383179, 3.095597267150879, 1.025941014289856, 0.08871252834796906, 0.5933730602264404, -1.1356878280639648, 1.6247607469558716, 1.8951197862625122, 0.8954790234565735, -0.2828352749347687, 0.5071044564247131, 0.12882722914218903, -4.168557167053223, -0.01647692546248436, 0.22008642554283142, 0.7617597579956055, -0.15631046891212463, -0.9999843835830688, -1.0825674533843994, -0.3406263291835785, -0.14495810866355896, -0.15251246094703674, -0.16527411341667175, 0.4935627579689026, 0.11348985135555267, 0.21071678400039673, -0.42517560720443726, 0.2238702028989792, -0.18235865235328674, -0.6243865489959717, -0.036438994109630585, 0.44909048080444336, -2.1629676818847656, -1.0060999393463135, 1.150234341621399, 0.37255313992500305, -0.4397583603858948, 1.814039945602417, -0.33306148648262024, -1.0740697383880615, 1.6049118041992188, -0.12033278495073318, -0.30230599641799927, -0.13402485847473145, 1.1955429315567017, -0.30323779582977295, 0.8951811194419861, -1.2992033958435059, -0.551694393157959, 1.348573088645935, -0.05544053763151169, -0.3845702111721039, 0.6032683849334717, 0.08176861703395844, 1.1716798543930054, -0.08741950243711472, 1.3646907806396484, 0.20107518136501312, 0.7036750316619873, -2.307990312576294, 0.06639658659696579, -1.0387767553329468, -0.5108854174613953, 0.26533493399620056, -1.7013564109802246, -0.3472846746444702, -0.167741060256958, 0.016058607026934624, 0.4312085807323456, 0.2677763104438782, -2.424795627593994, -1.2557214498519897, 0.9907416701316833, -0.6571822166442871, -2.5524497032165527, -0.3876826763153076, 1.0229947566986084, 0.5947022438049316, 0.195693239569664, 0.8039292693138123, 0.3101048171520233, -0.30042073130607605, -1.7879676818847656, 1.3236520290374756, 0.0008107910398393869, -0.12225563824176788, 1.7764017581939697, -0.04990209639072418, 0.05693887174129486, -0.13168728351593018, -0.4868920147418976, -1.9233386516571045, -0.047874368727207184, -0.9594648480415344, 0.05600468069314957, 2.837589740753174, -0.23100650310516357, -2.7435131072998047, -0.4536530077457428, 0.00713591231033206, 0.27758029103279114, -0.12771381437778473, 0.19816842675209045, 1.5500035285949707, 0.17125244438648224, -0.040483634918928146, -0.2835266888141632, 0.7999845743179321, -0.13188965618610382, 0.1602865606546402, 0.02097926288843155, 0.16788704693317413, 0.2698114514350891, -0.36744749546051025, -0.05028953775763512, -0.08598792552947998, 2.455737829208374, -0.19471217691898346, -0.9683264493942261, -0.467006653547287, 0.10166055709123611, -0.014667264185845852, 0.24440689384937286, 0.04627754166722298, -0.06188688799738884, 1.2158881425857544, -0.0289014782756567, 3.175725221633911, -3.0500409603118896, -0.375970721244812, 0.9167577624320984, -0.8830792307853699, 0.027031131088733673, -0.22176961600780487, 0.18603748083114624, -0.28963330388069153, -0.7399693727493286, -0.09748468548059464, 0.109375961124897, -1.8786288499832153, -0.024966515600681305, -1.4015202522277832, -0.06577027589082718, -0.902565598487854, 4.799386978149414, 1.368701457977295, 0.0766272321343422, -2.2033963203430176, 0.8850934505462646, 0.2682710289955139, -1.2529100179672241, -0.06138463318347931, 0.8535078763961792, 0.028588561341166496, -0.8258251547813416, 0.1804463118314743, 0.01533556915819645, -1.20625901222229, 0.8414249420166016, 0.6519867777824402, 0.20768769085407257, 0.02596822753548622, 2.038731813430786, -2.483785629272461, -1.0632479190826416, 0.5517961978912354, -0.71684330701828, 0.09275566041469574, 0.0037508371751755476, 0.7435833215713501, 0.09172862023115158, -0.7284039258956909, -0.16496695578098297, 0.9309265613555908, -1.5145381689071655, -0.6143402457237244, 0.12765580415725708, -0.545329749584198, 0.13117890059947968, 1.1456354856491089, 0.016361702233552933, -0.3921797275543213, 0.48358455300331116, -0.2993200719356537, 0.4931236505508423, -0.09587278962135315, 0.5965956449508667, 1.5129600763320923, -0.5478670001029968, 0.13363339006900787, -0.27095726132392883, 0.27734696865081787, 1.6946327686309814, -0.10342767089605331, -0.045052796602249146, -0.47023463249206543, -0.4380045533180237, -0.12730789184570312, -0.6421448588371277, -2.030574083328247, 1.4564224481582642, 1.579162836074829, -0.039541784673929214, -0.5610723495483398, -2.2843637466430664, -0.1942518949508667, 0.8973555564880371, 2.0925357341766357, -0.5568486452102661, -0.11287026852369308, -0.010294515639543533, -0.2807818055152893, -0.9511149525642395, -0.32801082730293274, 0.35694757103919983, 1.1178157329559326, 1.6946569681167603, 4.236677169799805, -0.0636516809463501, -0.392442524433136, 0.022745419293642044, -0.24896082282066345, 1.7603498697280884, -0.019758306443691254, 0.9784855246543884, 2.5571651458740234, -3.7435243129730225, -0.33927762508392334, 0.0006744880811311305, -0.1929457187652588, 0.9001620411872864, -0.08614592999219894, -0.7549598217010498, 0.003823405597358942, 0.06325104832649231, -0.29636436700820923, 1.0671437978744507, 0.10844768583774567, 0.263614684343338, -0.6466288566589355, 0.6786190867424011, -0.19578027725219727, 1.8433486223220825, 0.09497259557247162, -1.8350673913955688, 5.2561421394348145, 0.05453883111476898, 0.1228431686758995, 0.5195105075836182, 0.1030188575387001, 1.9543198347091675, 2.213566541671753, 0.10455866158008575, -0.04388438165187836, 0.36711612343788147, -0.022294027730822563, -0.23531022667884827, -0.2687731385231018, 0.11670901626348495, -2.9010369777679443, 0.03988121077418327, 0.6058362126350403, -0.15683764219284058, -1.4416413307189941, 0.357949823141098, 0.05831287056207657, 1.0480737686157227, 0.1648051142692566, 0.3856498599052429, -1.5871615409851074, 0.6734640002250671, -0.6572185158729553, 0.30314528942108154, 2.0393564701080322, 1.1467186212539673, -0.49965760111808777, -0.8251763582229614, 0.8528632521629333, -0.6128215789794922, 0.3629401624202728, -0.17931684851646423, -0.01940394751727581, 0.1516156941652298, 0.8975865244865417, 2.4708876609802246, 0.46578243374824524, -0.928354799747467, 0.4339451789855957, 0.30918559432029724, -0.14217984676361084, 0.5623654723167419, -0.01709621772170067, -1.8230711221694946, -0.01656484417617321, 0.18227145075798035, 0.8303154706954956, -0.10419534146785736, -0.271805077791214, -0.16375423967838287, -0.6961944699287415, 0.3317197561264038, 0.1625141054391861, 0.6011745929718018, -0.0029082968831062317, 0.020979803055524826, -0.650518536567688, -0.14624802768230438, 0.07216634601354599, 0.25485631823539734, 0.2929934859275818, 0.5869982242584229, -0.9585858583450317, 0.031120898202061653, -0.003192884149029851, -0.08468908071517944, 0.289772093296051, -1.181946039199829, -0.29210934042930603, 0.712761402130127, -0.5357728600502014, -0.9878578186035156, 0.6294951438903809, -0.07518376410007477, -0.4894394278526306, 0.9336721897125244, -1.1306551694869995, 0.14427661895751953, 0.07022136449813843, -0.3327222466468811, -0.8322756886482239, -0.7699688076972961, -0.22387003898620605, 2.6380245685577393, -0.47728559374809265, -0.21725085377693176, 0.07194105535745621, -0.021680673584342003, 0.3874833881855011, 0.11796692758798599, 0.858853280544281, -1.6080878973007202, 0.845952570438385, -0.10360004007816315, 0.9445975422859192, 5.8609514236450195, -1.1110883951187134, -0.1096775233745575, -0.3687448501586914, 0.20247027277946472, 2.1373133659362793, -0.0219980888068676, 0.03757292777299881, -0.5955619215965271, -0.1435965895652771, 0.5727738738059998, 0.5565298199653625, -1.5259863138198853, 0.8437738418579102, 0.10060163587331772, 0.010151287540793419, 1.9357646703720093, 0.06569170951843262, -0.10005155205726624, 0.3255425989627838, 1.3225764036178589, -3.003533363342285, 0.02624478004872799, -0.09547574073076248, 1.3775339126586914, -0.22783684730529785, -1.5033031702041626, 0.03403226286172867, 1.2835068702697754, -0.3214920461177826, 0.232060968875885, -1.0402719974517822, -0.21305474638938904, -0.24466721713542938, 0.14967671036720276, 0.8232829570770264, -0.10515225678682327, -0.4004271924495697, -2.3664989471435547, 0.6346789598464966, 0.14001861214637756, 0.8574247360229492, -1.6440659761428833, -3.406569480895996, -0.026555707678198814, -0.13372865319252014, 0.42433711886405945, -0.6006458401679993, -1.6549906730651855, -0.15238596498966217, 1.301888108253479, 0.004264612216502428, 0.7153674960136414, 0.6790817379951477, -0.07552220672369003, 0.16102340817451477, 0.22974149882793427, 0.506658673286438, -0.5626471638679504, -0.11378451436758041, -0.0038735975977033377, -0.07903330028057098, -0.21552231907844543, 1.1654274463653564, 1.3241041898727417, 0.1375661939382553, -0.12928342819213867, -0.3449147045612335, -1.7368390560150146, -0.26882287859916687, -0.20722056925296783, 0.13910607993602753, 3.5520918369293213, -0.6862249374389648, 0.040546808391809464, -0.10722570866346359, -0.34389546513557434, -0.14292453229427338, 0.035262398421764374, -0.7676297426223755, -0.0008487461018376052, -1.5496543645858765, 0.1575554758310318, 0.15766692161560059], [0.26625242829322815, -1.4514884948730469, 0.33006542921066284, -0.47641241550445557, -0.6508572101593018, -0.36405766010284424, -0.010648884810507298, -2.3732125759124756, -0.014338962733745575, -3.040566921234131, -1.8558483123779297, -0.11791764944791794, 1.0911856889724731, -0.12403273582458496, 0.1798337697982788, -0.834626317024231, -0.8280606269836426, -0.3545096218585968, -0.36038196086883545, 0.5914273858070374, -1.0087085962295532, -0.7304401993751526, -0.05820591002702713, 0.4804213345050812, 0.13127285242080688, 0.4176484942436218, 0.343296617269516, 0.4892314076423645, 0.7563382387161255, 0.02502850815653801, -0.575110912322998, 0.03798411041498184, 0.48811882734298706, 0.14798474311828613, -0.041661858558654785, 0.7395538091659546, 0.04841678589582443, -0.2435358315706253, 0.11016587913036346, 1.160914421081543, -1.0073823928833008, 0.8744612336158752, 0.04059980437159538, -0.13723157346248627, 0.1288190633058548, -0.5825942158699036, -0.4356852173805237, -0.29228952527046204, 1.0728627443313599, 0.41865649819374084, 0.4812486171722412, -0.17603856325149536, -0.014687823131680489, -0.14531825482845306, 0.165829598903656, -0.3565790355205536, 0.07538701593875885, 0.002438388066366315, -0.02946350909769535, -0.027283603325486183, 0.04208408296108246, 0.18453659117221832, -0.06244036927819252, 0.2907235026359558, 1.0722627639770508, 0.3610266447067261, -0.34764862060546875, -1.3078371286392212, -0.3408370316028595, -0.2520671486854553, -0.2905220091342926, 0.49170100688934326, -0.6043076515197754, -0.05501571297645569, -0.3711567223072052, -0.2430218607187271, -0.08057405799627304, 1.4275212287902832, 0.0062808627262711525, -0.3934175372123718, -0.5967709422111511, -0.38238292932510376, 0.17187345027923584, -0.23169514536857605, 0.24483688175678253, -0.05641476437449455, -0.1717413067817688, 0.7011610865592957, -1.3681570291519165, -0.30373895168304443, 0.06889797747135162, -0.6149155497550964, -0.015428912825882435, 0.7057108283042908, -0.5487269163131714, -0.5476524233818054, 0.1579643040895462, 0.7709022760391235, 1.0617151260375977, 0.05123138427734375, -0.09254250675439835, 0.39586329460144043, -0.21402998268604279, 0.02039981447160244, -0.3158213496208191, -0.01257549598813057, -0.0963623970746994, 0.1408710777759552, 3.5340442657470703, -0.19827713072299957, -0.34931525588035583, -0.04202759265899658, 0.32354122400283813, -0.19389031827449799, -0.6155890822410583, 0.6266132593154907, 1.2425466775894165, 1.3046154975891113, -0.4675581157207489, -0.7574622631072998, 1.1050423383712769, -0.4072801470756531, 0.19779019057750702, 1.2502176761627197, 0.04892931133508682, 0.8796302080154419, 0.8662019968032837, -0.09255670756101608, -2.030776023864746, -0.535054624080658, 3.279283285140991, 1.0172898769378662, -0.06518785655498505, 0.6996212601661682, 1.6146888732910156, 0.25588610768318176, 0.31390947103500366, -0.3977218270301819, 0.34144455194473267, -0.7418277859687805, 0.2469283640384674, 0.0576513335108757, -0.10151608288288116, 0.036353252828121185, -2.32380747795105, -1.8044673204421997, -0.30428776144981384, -1.088950753211975, -0.023043038323521614, 0.007747326977550983, -0.09911096841096878, 0.008013087324798107, -0.315506249666214, 0.20810143649578094, 0.3904933035373688, 0.7294341921806335, 1.265291690826416, -0.2503734827041626, 0.48187577724456787, -0.01980459690093994, 0.7823611497879028, 0.010000253096222878, 0.007814192213118076, 1.2957475185394287, -0.16564629971981049, -0.01759733073413372, -0.3070743680000305, -1.7011401653289795, -0.025149693712592125, -0.057130638509988785, 0.7681284546852112, -0.2966252565383911, 0.655383288860321, 1.637300968170166, 0.05059226602315903, -0.06821228563785553, -0.14350788295269012, -0.2595808506011963, 0.39852768182754517, -0.1566479653120041, 0.5336571931838989, 0.027329450473189354, -0.001701430999673903, -0.1804686039686203, -0.5151992440223694, 3.311330556869507, -0.5654655694961548, 0.017691584303975105, -0.088912732899189, -0.044533148407936096, 0.13485775887966156, -1.726987600326538, 0.41793590784072876, -1.4556841850280762, 0.7170329689979553, 0.34704703092575073, 1.115798830986023, 0.2761450409889221, -2.3580143451690674, 1.0009199380874634, 0.2621171176433563, 0.1533600240945816, -0.18079228699207306, -0.04386594891548157, -0.08715341240167618, -0.08999036252498627, 0.4761750400066376, 0.17922495305538177, 0.5986790060997009, 0.09380724281072617, -0.20907439291477203, 0.4724295139312744, 0.0002461133699398488, 0.06212892010807991, 0.3462960422039032, 0.2448420226573944, 1.1739799976348877, -0.8240088224411011, -0.02249942719936371, -0.25334256887435913, -1.2398236989974976, 2.0597431659698486, -0.9620097875595093, 0.46883463859558105, 0.1366548091173172, 0.035263560712337494, 0.6272795796394348, -1.1132912635803223, 1.4675458669662476, -0.06622112542390823, 0.8850871324539185, -1.060394525527954, 0.03449096158146858, -1.6523512601852417, -1.7955154180526733, 0.14670641720294952, 0.018702983856201172, 0.4490191340446472, 0.08456391841173172, -0.0008238096488639712, -0.31801334023475647, -0.9119988679885864, 0.06407254189252853, 0.6773494482040405, -0.4695684015750885, 0.24791979789733887, 3.6551671028137207, -2.7702202796936035, 0.013137413188815117, 0.027007874101400375, 0.2028099149465561, 0.4167305529117584, -2.237109899520874, -0.1896461397409439, -0.7425587177276611, 0.14718948304653168, -2.6296679973602295, -1.2147431373596191, -0.009029178880155087, 1.2853082418441772, 0.5030722618103027, -1.6268571615219116, 1.0268900394439697, 2.753458023071289, 0.053526390343904495, -0.7878457307815552, 0.13666242361068726, 0.7738023400306702, 0.7503716349601746, -1.6398723125457764, 0.0021477104164659977, 1.378739833831787, 0.2028363198041916, -0.4214726686477661, 0.30475229024887085, -0.1793706864118576, -1.2229050397872925, 0.11490616947412491, -0.05635213851928711, -2.2909629344940186, 0.0036774897016584873, -0.08944886177778244, 0.6373159289360046, -0.4167102575302124, -0.1518576443195343, 0.1906217783689499, 2.4891302585601807, 1.0852346420288086, -0.1086859256029129, -1.2606464624404907, -0.04543965682387352, 0.6126688718795776, -1.1237200498580933, 1.7396306991577148, -0.8348652124404907, -1.5432120561599731, -0.06099013611674309, 0.006598962005227804, -0.7518858313560486, -0.15909677743911743, 0.08599065244197845, -0.015856750309467316, -0.11343861371278763, 0.13860754668712616, 1.0298584699630737, -0.8569160103797913, -0.17465022206306458, -3.6042098999023438, 0.6085938215255737, 0.5041964650154114, 0.1576615273952484, -1.1336809396743774, -0.19264477491378784, 0.3282674551010132, -1.5615750551223755, -0.8849531412124634, 0.8565363883972168, -0.10022031515836716, 0.09896964579820633, -0.025536086410284042, -0.1096002608537674, 0.0872081071138382, 0.11177673190832138, 0.5338734984397888, -0.31638529896736145, 0.6941848993301392, 2.297912120819092, -2.289496421813965, 0.6339157223701477, 1.6209601163864136, 0.1197303906083107, 1.273766040802002, -0.053367894142866135, 0.016794921830296516, 0.3274533748626709, 0.70365309715271, 0.26195406913757324, 0.7254837155342102, -0.5959914326667786, 0.1007765382528305, -0.9943689107894897, 0.15404103696346283, -1.1250572204589844, 0.4148378372192383, 0.8764658570289612, 0.855460524559021, 0.23025493323802948, -0.09464430809020996, -0.12249858677387238, -0.07176262140274048, -0.3427509069442749, 0.1452013999223709, -0.4880901873111725, 0.3015538156032562, 0.1600545346736908, -1.1447840929031372, 0.17124556005001068, -0.8598944544792175, -2.3200058937072754, 0.5461011528968811, 0.7608125805854797, 0.5244150757789612, -0.4735105633735657, -0.1828118860721588, 0.6491220593452454, 0.337939590215683, -0.13415609300136566, 0.537786602973938, -0.42061254382133484, -0.9343833327293396, 0.10216891765594482, -0.21241626143455505, -0.13899953663349152, -2.2604613304138184, 0.17237313091754913, 1.5731420516967773, 0.6305150389671326, 0.17397278547286987, 2.341829299926758, 0.8174958229064941, -0.85540372133255, -0.29168927669525146, -0.0023900200612843037, 1.6579021215438843, 0.3993302881717682, 0.13110661506652832, -0.0036944583989679813, -0.10243252664804459, 2.3143794536590576, -0.5763717293739319, -0.5243086218833923, 0.03910144418478012, 0.6087137460708618, 0.1278572827577591, 0.6957307457923889, -0.3164900541305542, -0.4349140226840973, -0.14639230072498322, 1.6198463439941406, -0.0004374182317405939, 0.21477897465229034, -1.1402374505996704, 0.33888015151023865, -1.330338954925537, 0.0034637872595340014, 0.6696340441703796, -2.8433480262756348, 0.24779805541038513, -0.5719510912895203, 0.37993600964546204, -0.3978417217731476, -2.2212002277374268, 1.2911680936813354, -0.6973922848701477, 0.02198627032339573, -0.24775458872318268, -0.20246627926826477, 1.1606251001358032, 0.03157814219594002, 0.585614800453186, -0.08903622627258301, -0.6869317293167114, 0.5378016233444214, 0.5849777460098267, -1.4099597930908203, -0.5060970187187195, 0.015054619871079922, -0.4174444377422333, -0.3268975615501404, 0.1916213184595108, 0.7682462930679321, -0.04970429465174675, 0.42995962500572205, -1.870194673538208, -2.436525583267212, -0.7554440498352051, -0.44890573620796204, -0.03036338835954666, -0.08905193209648132, -0.18758605420589447, 1.3069185018539429, -0.5712727904319763, -0.15186931192874908, 0.2587294578552246, 0.04923563823103905, -0.23030009865760803, 1.3377267122268677, -1.4185428619384766, 0.3801976144313812, -0.006356086581945419, -1.0096280574798584, 0.779075562953949, -0.00533120334148407, -0.28563064336776733, 1.334768533706665, -0.38903895020484924, 0.10519105941057205, -0.7196869254112244, 1.1507660150527954, -0.006641782354563475, 0.3788801431655884, -0.03074587509036064, -0.021761296316981316, 0.5457934141159058, -0.009194187819957733, 0.5228786468505859, -2.906649112701416, 0.04816789552569389, -0.9178090691566467, 0.7470901012420654, -0.7570104598999023, -0.6104227900505066, -0.4255238473415375, 2.2703638076782227, 0.0777728334069252, -0.5238549113273621, -0.20744562149047852, 0.5559762120246887, 0.07280305027961731, -3.599907636642456, 0.254778653383255, -0.29212793707847595, -0.7890300154685974, 0.1759529560804367, 0.3414456248283386, -2.0269815921783447, -0.950194239616394, -0.7937048673629761, 2.8013112545013428, 0.0026559417601674795, -0.48424801230430603, -1.4391381740570068, -1.093966007232666, -1.0079551935195923, 0.006312067154794931, 0.7576101422309875, 1.1346789598464966, -0.05695819482207298, 4.259246826171875, -0.4214843213558197, 0.1226947233080864, -2.9750583171844482, -0.6755145788192749, -0.0072897071950137615, 1.7212671041488647, -0.02300107292830944, -0.08417321741580963, -0.8222516179084778, 0.02336961403489113, -1.2132798433303833, 1.3887311220169067, -0.13055773079395294, 0.11566969007253647, 0.11491310596466064, 1.1699650287628174, -0.9751307964324951, -3.1532227993011475, -0.43638575077056885, -1.09304940700531, -0.6095516681671143, 0.22098201513290405, -0.3080180585384369, -0.7838158011436462, 0.4944024980068207, -0.7939140200614929, -0.25168028473854065, 0.7586997747421265, -0.40119871497154236, -0.3266292214393616, -1.4147957563400269, 0.25417259335517883, 2.033245801925659, -0.10121504962444305, -0.021777333691716194, 0.6169325113296509, 1.377630352973938, 0.4739930033683777, -0.21563419699668884, 2.808047294616699, -0.7473836541175842, 0.09103092551231384, -0.00022664052085019648, 1.0411243438720703, -0.3254668414592743, 0.8837113380432129, -0.033347759395837784, 0.3229048550128937, -0.008208860643208027, -0.07682710886001587, 0.027033202350139618, 0.22272297739982605, 0.13791997730731964, 0.44345778226852417, -0.309457927942276, -0.5744603276252747, -1.3334300518035889, -0.14657345414161682, -0.5989713668823242, -0.07462915778160095, -0.40692031383514404, -0.6426035761833191, 0.8509197235107422, -0.0419737733900547, -0.9387386441230774, -0.7409428954124451, -0.4867662489414215, 0.025504518300294876, -1.2698187828063965, -1.522692084312439, 1.7438223361968994, -0.03863145411014557, -1.3178654909133911, -0.15046806633472443, -0.7064318656921387, 0.7690038084983826, 0.8457061648368835, 1.8688135147094727, 0.19619934260845184, -1.9271620512008667, -0.0912264809012413, -1.7028030157089233, -0.5918825268745422, -0.048751842230558395, -0.017921950668096542, 0.10592062771320343, 0.5500181913375854, 0.10225285589694977, -0.06056657060980797, 0.4069552421569824, -0.2846958637237549, 0.6484431028366089, -0.012565390206873417, 0.23643502593040466, -0.13468243181705475, 0.003363091964274645, 0.020406994968652725, -0.7936285138130188, -0.5963457822799683, -0.7935733795166016, -0.19243231415748596, 0.5418593287467957, -0.09378349781036377, -0.25254541635513306, -0.8440887331962585, 0.07335589081048965, -1.3894838094711304, -0.25391122698783875, 0.7602152228355408, -0.12011420726776123, -0.02852240577340126, -1.5287675857543945, 0.5855281352996826, -0.6938577890396118, -0.5214187502861023, -0.23931078612804413, -0.36068663001060486, -0.3712392747402191, 0.3584228754043579, 0.23054690659046173, 0.759606659412384, -0.0070631797425448895, 0.08975253254175186, 0.2562221586704254, 0.29135647416114807, -0.1261494755744934, 0.9016506671905518, 0.7661095261573792, 0.34416717290878296, 0.11906815320253372, 0.015119875781238079, -0.7707549333572388, -0.16540086269378662, -0.008114238269627094, -0.10692574828863144, 2.0842835903167725, -0.23112574219703674, -0.0027473014779388905, 1.3411049842834473, -0.02524142898619175, -2.396466016769409, -1.1353275775909424, 0.15657924115657806, 1.2230697870254517, 0.7773164510726929, 0.08702966570854187, 1.0985723733901978, -1.1785403490066528, 0.20443537831306458, 1.7626134157180786, 0.07350456714630127, -0.9881243705749512, -0.4388618469238281, 0.14455120265483856, 1.4420616626739502, -0.7382456660270691, 0.014885349199175835, 0.0043464903719723225, -0.7306401133537292, -1.1812306642532349, -0.7434703707695007, 0.13198940455913544, 0.4631572663784027, 0.16482265293598175, -0.05172460898756981, 1.1579396724700928, -0.03692285344004631, 1.4853707551956177, -0.9680122137069702, -0.8981420993804932, -0.14052072167396545, 1.3425105810165405, -0.17806744575500488, -0.5364654064178467, 2.0758755207061768, 0.24944929778575897, 0.5213451385498047, -0.8536968231201172, -0.05824930965900421, 0.6687169671058655, 0.00573853962123394, 0.10623114556074142, -0.03945140168070793, -0.3470287621021271, 0.26956838369369507, 0.1717885434627533, -0.4183293581008911, -0.04905879497528076, -0.013301591388881207, -0.7952509522438049, -0.5271410942077637, -0.8865111470222473, -0.2197352796792984, -0.07300049811601639, 0.560256838798523, 1.4595571756362915, 0.48657867312431335, -0.4908111095428467, 0.05881965160369873, -5.711819171905518, -0.2391788512468338, -0.0035307209473103285, 0.8801862001419067, -0.448486328125, -0.10769354552030563, -0.0768555998802185, 0.8709641098976135, 0.03691619262099266, -0.08321215957403183, 1.510833740234375, 2.0064291954040527, -0.045827094465494156, 0.642927348613739, 1.4804719686508179, -0.36063531041145325, 0.057710397988557816, -0.8261143565177917, 0.19429542124271393, 0.11583906412124634, 0.06518412381410599, 0.8497766852378845, -0.7039378881454468, 0.26600226759910583, -2.220456123352051, 1.6401864290237427, 0.428182452917099, -0.6264064908027649, -0.02016138657927513, -0.3678232729434967, 0.2890753149986267, -0.6687425374984741, 0.14177341759204865, 0.3350532352924347, -1.2429726123809814, 0.04967323690652847, 0.5061588883399963, -0.4709874987602234, -0.029225369915366173, 0.4015776216983795, 0.013165968470275402, 0.4077998101711273, -0.0027135293930768967, -0.5489639639854431, 0.041518133133649826, 0.16803637146949768, 0.22288967669010162, 0.2889584004878998, -0.0007767744828015566, 0.08041805028915405, 1.1841026544570923, 0.1435050368309021, 0.4519518315792084, 2.1596522331237793, 1.2795668840408325, 1.2975937128067017, 1.1893219947814941, 0.1390228271484375, 0.22767497599124908, -0.02023397758603096, 0.25291693210601807, -0.005148048512637615, -1.9855090379714966, 0.04949011653661728, 0.11260472238063812, 0.1472722738981247, -0.5276975035667419, -1.3498233556747437, -0.2346612811088562, 0.25779327750205994, 0.16072072088718414, -0.0671868771314621, -0.0030760590452700853, -1.018147349357605, 0.060934878885746, -0.01911897584795952, -0.006288761738687754, 0.035595450550317764, 0.277014821767807, -0.7146056294441223, 0.23165883123874664, 0.2437354475259781, -0.016229471191763878, 1.1931484937667847, 0.12605717778205872, -0.6772326231002808, 0.10046622902154922, 0.014371635392308235, 0.9451697468757629, 0.12302184849977493, -0.9740729331970215, -2.5116055011749268, -0.4427568316459656, -0.6241163611412048, -0.40487968921661377, 0.27575773000717163, 0.0009086196660064161, 0.06570667028427124, 0.03144073113799095, -0.08192800730466843, 0.2094132900238037, 1.1287490129470825, 1.3953909873962402, -0.12640100717544556, 0.07888439297676086, 0.22349153459072113, -2.821847677230835, -1.977548360824585, -0.2873351573944092, -0.7629261612892151, 0.5785372257232666, -0.30467846989631653, -1.0865540504455566, -0.00438669603317976, -0.8993476629257202, -0.2575235068798065, 0.1698015034198761, 0.02871701493859291, -0.3815988302230835, -0.10365819185972214, -2.5954904556274414, -0.34397467970848083, -0.379869669675827, -0.21433572471141815, -4.66120719909668, -0.2709457576274872, 3.0491106510162354, 0.12013573944568634, -0.011501889675855637, -0.8331522941589355, -0.037299662828445435, 0.3563846945762634, -0.5845499634742737, -0.0828423723578453, 0.8382426500320435, 0.20115533471107483, 3.0029587745666504, 1.622786045074463, 0.030010338872671127, 0.28458961844444275, -2.4477028846740723, 0.10952005535364151, 0.49763041734695435, -1.090368390083313, -0.6891679167747498, -0.3944392204284668, 0.5073718428611755, -0.8513884544372559, 1.6432292461395264, -0.6123557686805725, 0.01718766987323761, -0.20570087432861328, 0.0153683265671134, -1.0701119899749756, -1.4578913450241089, 0.14719875156879425, 0.019375260919332504, -2.3406028747558594, -0.016693606972694397, 0.6039669513702393, 0.0052725644782185555, 1.208299994468689, -1.158772349357605, 0.854414165019989, 0.6930453777313232, -0.0026651679072529078, 0.22871793806552887, -0.0250334944576025, 0.2783606946468353, 0.14885413646697998, -0.7692673802375793, -0.04732927307486534, -1.0898550748825073, -0.26233187317848206, -0.10446469485759735, 0.014776312746107578, 0.18241341412067413, 0.17674124240875244, 0.0022183996625244617, 0.6796203255653381, 0.23393747210502625, -0.9707956910133362, -0.12956666946411133, -0.585911214351654, 0.7636549472808838, 0.30711665749549866, -0.25559544563293457, 0.0622287355363369, 0.21837052702903748, 0.00168312294408679, -0.2098103165626526, 1.4339215755462646, 0.6024283170700073, 0.32176926732063293, 0.01792866736650467, 1.019412875175476, -1.9055649042129517, 0.04581501707434654, 0.5359112620353699, -0.3947843015193939, 0.40399590134620667, -0.2805471122264862, -0.7789965867996216, -0.5541480779647827, -2.870283365249634, 0.09199479222297668, -0.39219218492507935, 0.3513699471950531, -0.007220805156975985, -1.4676059484481812, 0.2365184724330902, 0.230296328663826, -0.2604704201221466, 0.17358699440956116, 0.2306644320487976, 0.36964017152786255, 0.30342337489128113, 0.6697277426719666, 3.319946765899658, -2.0042600631713867, -0.03777788206934929, 3.2770841121673584, -0.11717230081558228, 0.9101974368095398, -1.2991931438446045, 0.1642252653837204, 2.685563802719116, -0.004909399431198835, -0.6567093729972839, -1.189579963684082, 0.5489864349365234, 0.46768975257873535, 0.5531575083732605, 0.14237630367279053, -0.12540069222450256, -2.334310293197632, 0.1067434698343277, -2.0666933059692383, 0.3232620358467102, -2.814845323562622, 0.8850654363632202, 0.10628858953714371, -0.05264995992183685, -0.12263482064008713, 0.6819946765899658, -1.4261976480484009, 0.3136819005012512, 0.039993222802877426, -0.20167326927185059, -0.13098116219043732, -0.2550264596939087, 0.8551990389823914, 0.29612964391708374, 0.4997185170650482, 0.3821116089820862, 0.43487614393234253, 0.3316226005554199, -0.47116944193840027, 0.8067272305488586, 0.2215113490819931, -0.054952047765254974, -0.18405653536319733, 2.0350282192230225, 0.5052915811538696, -3.1953136920928955, 0.4973842203617096, -2.432447910308838, -0.7874554395675659, 0.15477991104125977, -0.014729038812220097, 0.7228193879127502, 1.7601888179779053, -1.7416974306106567, -0.9252173900604248, -0.20573697984218597, 0.008790699765086174, -0.43734583258628845, -0.04735977202653885, -0.7453420758247375, 0.4343203604221344, 1.5035401582717896, -0.15174199640750885, -1.7805700302124023, 1.9430171251296997, 0.07743387669324875, -0.0019021229818463326, 0.30706048011779785, -0.10367648303508759, 2.294198989868164, 0.022159213200211525, -0.01766270212829113, -0.27272114157676697, 0.018584270030260086, -0.7496564388275146, 0.026604795828461647, 2.916546106338501, -0.01305516716092825, -0.43665897846221924, -2.9462549686431885, -2.1957013607025146, 1.3803273439407349, 0.6645316481590271, -1.8102786540985107, -0.22051319479942322, 0.016386928036808968, -0.06404844671487808, 0.39507806301116943, 0.12850698828697205, -0.4165806174278259, -2.519322633743286, 0.06022963300347328, -1.4122523069381714, -0.13435953855514526, 0.6878131628036499, -1.5054341554641724, 1.9893429279327393, -0.4835410416126251, 0.3373488485813141, 0.24088600277900696, -0.5456951856613159, 0.34201958775520325, -0.053027525544166565, 0.9914954900741577, -1.8113651275634766, -0.010341232642531395, -0.000684400787577033, 0.0693577378988266, 0.040103405714035034, -0.4421278238296509, 0.10997510701417923, -0.25010746717453003, -0.14528749883174896, -0.07517953962087631, -1.2700307369232178, 0.263264536857605, 1.5904433727264404, -0.7329905033111572, 2.0744028091430664, -0.08770562708377838, 0.7438676953315735, 0.11159703135490417, 0.0015725394478067756, -0.019625728949904442, 0.531466543674469, 0.10736416280269623, -1.8596324920654297, 0.0912928357720375, 0.41949784755706787, 0.386313259601593, 0.1722276210784912, 0.09176576137542725, 1.022592544555664, -0.44694504141807556, -0.14579930901527405, -0.4970449209213257, -0.6401412487030029, 0.3767484724521637, -2.0724098682403564, -0.11757460236549377, -0.19544699788093567, 0.5183553099632263, -0.07403815537691116, -1.535617709159851, -0.423780620098114, -0.9658960103988647, 0.8979681134223938, 1.0726256370544434, -0.17625947296619415, 0.6044894456863403, 0.24457818269729614, 1.0267963409423828, 0.016889886930584908, 0.2788448929786682, 0.39136672019958496, 0.10315876454114914, 0.8653661608695984, 1.2086173295974731, -0.06216329336166382, 0.3406616151332855, -0.3935166299343109, -0.21301376819610596, 0.17205004394054413, -2.936741352081299, -0.09781134873628616, 0.3873079717159271, 0.29352906346321106, -0.3327178359031677, 0.2949525713920593, -3.485229730606079, -0.07136591523885727, -0.9879306554794312, -0.4432700276374817, -0.5167787075042725, 0.33376264572143555, -0.2928740084171295, -1.9779940843582153, 0.008618194609880447, 0.07687340676784515, 1.029815673828125, 0.03383520990610123, -0.4174964129924774, 0.12283103913068771, -0.026284554973244667, 0.314628541469574, -0.26146385073661804, 0.14849039912223816, -0.3224097490310669, -0.05903615057468414, 0.9775153398513794, -0.06839194893836975, 1.7436068058013916, 0.19560864567756653, -1.4999040365219116, -0.03529990091919899, 0.2532349228858948, 0.17619702219963074, -0.8539533615112305, 2.1223716735839844, -0.044159285724163055, 0.6301434636116028, -0.27249428629875183, -0.31523287296295166, -0.08433658629655838, 0.20328080654144287, 3.582162618637085, 0.20486530661582947, 1.251141905784607, -0.8200305104255676, 0.28426098823547363, 0.02452600933611393, -2.206435203552246, -0.1778360903263092, -0.057872213423252106, -0.02485036663711071, -0.46967869997024536, 0.005768647417426109, -0.38435816764831543, 0.2385885864496231, 0.37983253598213196, -0.015284267254173756, -0.0735180675983429, -0.4561409652233124, -0.2607179880142212, 0.9391360878944397, 0.002089665038511157, 0.42176172137260437, 0.6466878652572632, -0.19573242962360382, 0.08233506977558136, -0.0478060282766819, -0.7398826479911804, -0.0245354026556015, -0.16098761558532715, -0.14201723039150238, 0.4403958320617676, -0.08605261892080307, -0.4383159279823303, 3.3792903423309326, 0.19469605386257172, 0.047541722655296326, 0.006151745095849037, 0.434709370136261, -0.053354669362306595, 1.318947672843933, -0.9001011252403259, -0.07866299897432327, 1.2240259647369385, 1.3701938390731812, -0.6800014972686768, 1.3724288940429688, -0.03931811451911926, -0.06594326347112656, 0.19862821698188782, -0.9957395195960999, -0.1313098520040512, -0.08085649460554123, 0.3106784522533417, 0.5191212892532349, 1.1093072891235352, -0.24226047098636627, -0.29225677251815796, 0.44569462537765503, -0.04982965812087059, -0.12799100577831268, 0.7214842438697815, 0.7816829085350037, 0.03915608674287796, -0.9844236373901367, -0.09272657334804535, -2.151169538497925, 0.008564244955778122, -0.11272402852773666, 0.5097837448120117, 0.08097483217716217, -0.07457855343818665, 0.0017307187663391232, 0.0551394447684288, -0.22212442755699158, -0.22146065533161163, 1.5395874977111816, -1.3768452405929565, 0.1188676580786705, -0.06553290784358978, 0.9159611463546753, -1.4450212717056274, -0.2845192551612854, 0.09392103552818298, -0.08970553427934647, -0.3991556167602539, 0.08177778869867325, -0.05201494321227074, 0.2658809423446655, -0.07168392091989517, 0.005639541894197464, -1.0385382175445557, 0.9035502672195435, 0.18061722815036774, -1.5456441640853882, -0.7282184362411499, 0.35317644476890564, -0.019272230565547943, 0.24985966086387634, -0.020604824647307396, -0.04249769076704979, -0.3056229054927826, -0.38656750321388245, -0.1594269722700119, -3.2341840267181396, -0.19683118164539337, 0.05622577667236328, -0.41366228461265564, 0.4075930416584015, 0.535211980342865, 1.093774437904358, -3.1264474391937256, -2.6045939922332764, 0.12726320326328278, 0.4097133278846741, -0.43354031443595886, 0.07628922909498215, 0.030970200896263123, -0.5026669502258301, -1.6910936832427979, -0.2991809546947479, -0.3831522762775421, 0.3335800766944885, 0.2112639844417572, -0.1236601397395134, 0.6726447939872742, 0.5616906881332397, -0.8433979153633118, 0.7159025073051453, 0.6525112390518188, 0.6101194024085999, 0.46390312910079956, 1.0478488206863403, 0.20188000798225403, 0.533758282661438, 0.4696251153945923, 0.33101579546928406, 1.531385898590088, -0.2769446074962616, 0.04859524592757225, -0.18897512555122375, 0.6518130898475647, 0.4274793863296509, 0.4723634421825409, -1.1517256498336792, -0.3233325779438019, -2.2528634071350098, 0.28788453340530396, 0.013246426358819008, -0.23995020985603333, 3.5324602127075195, -0.12581591308116913, 1.9554890394210815, 0.6313679814338684, -0.5836160182952881, -0.323003888130188, 0.06121763959527016, -2.3682093620300293, 0.2804039716720581, -2.655233144760132, -0.8930435180664062, 0.3053476810455322, 0.06883399933576584, -0.0057904901914298534, -0.06206435337662697, -0.36700311303138733, 0.39722180366516113, -0.2773488461971283, 0.7445272207260132, 1.7681293487548828, 0.047485385090112686, 1.164638876914978, -0.08893776684999466, 0.022997794672846794, -0.46348607540130615, 0.08527776598930359, -0.49565938115119934, -0.7025692462921143, -0.4152035415172577, -0.12912651896476746, -0.5969762206077576, 0.7534369230270386, 1.9170057773590088, 0.2694219946861267, -0.9492745399475098, -0.21305064857006073, -0.16720448434352875, 0.41184109449386597, -0.0772673711180687, -0.05377134308218956, 0.19802790880203247, 0.1771026849746704, -2.5833473205566406, 0.023939907550811768, 0.028262846171855927, 0.12881438434123993, 0.04528828337788582, -1.139399766921997, -0.799680233001709, -0.32975688576698303, 0.3677104115486145, -0.8129621148109436, -0.5205907225608826, 2.8611867427825928, 0.6941761374473572, -0.5502798557281494, -0.230977401137352, -0.8486781120300293, -0.4217489957809448, -0.5941349864006042, -0.44230303168296814, 0.03464217483997345, 0.428933709859848, -0.7550837993621826, 0.9838215708732605, -0.3139120936393738, 0.10004326701164246, 0.22476349771022797, 0.6998482942581177, 0.005358169320970774, 0.6131539940834045, -0.3029997944831848, 0.11063271760940552, 0.10547538101673126, 2.1405632495880127, -0.6496694087982178, -0.5180934071540833, 1.452073335647583, 0.006445952225476503, -0.18755634129047394, 0.12356597185134888, -1.6510567665100098, 0.34594935178756714, -0.0275413915514946, -0.7626998424530029, -0.8702400326728821, -0.14644427597522736, 3.2163097858428955, 1.0977391004562378, -0.7724447250366211, 1.240965485572815, -0.07884566485881805, 0.5709984302520752, 0.24753738939762115, 0.4623202383518219, 0.33163416385650635, -2.4489405155181885, 0.046783581376075745, -0.0393596775829792, -0.6883989572525024, 0.21905797719955444, -0.1055169627070427, 0.00666792830452323, -1.0435669422149658, -0.9015982151031494, -0.02400873228907585, 0.3043450117111206, 0.6141251921653748, -0.6882694959640503, 0.15607915818691254, 1.489829421043396, 0.03888874873518944, -0.07721349596977234, -0.6361842751502991, -1.3748703002929688, -2.2059836387634277, -1.0738073587417603, 0.34518367052078247, 1.445410966873169, -0.46251484751701355, -1.0276106595993042, 0.7652801871299744, 0.1384827196598053, 0.4974174499511719, 1.6818650960922241, -0.06250573694705963, -0.047905758023262024, -0.23137837648391724, -0.07578589767217636, 0.016069462522864342, 0.23700392246246338, -0.10132311284542084, 0.3412218391895294, 0.1320800632238388, 0.3035335838794708, -0.34709036350250244, -0.3325190544128418, 0.35365939140319824, 0.21708513796329498, -2.521176338195801, -0.04220738634467125, -0.6493869423866272, -0.9929285049438477, 0.046121083199977875, 0.6940751671791077, -0.20627360045909882, -0.05729839578270912, -0.8208566904067993, -0.7301091551780701, -0.5271459817886353, 1.5565903186798096, -0.19951607286930084, 1.4340929985046387, 0.5132623314857483, 1.1610474586486816, 1.60370934009552, -0.02523929625749588, -0.9127786755561829, -0.005945811979472637, 1.0248651504516602, 0.6045228242874146, -0.7104814052581787, -0.14144013822078705, 1.4557108879089355, -0.13730299472808838, -1.1772377490997314, -0.05074307322502136, -0.2346833199262619, -0.445595920085907, -0.1727752983570099, 0.26629945635795593, 1.2149485349655151, 1.7437337636947632, -0.10641689598560333, 2.2519631385803223, 0.1968267560005188, -0.3099345266819, 0.9075868725776672, -2.3418445587158203, -1.3116506338119507, 0.1981457620859146, -2.1930301189422607, 0.37886843085289, 1.5495257377624512, 0.038908738642930984, -0.7557891607284546, 0.5801390409469604, -0.5021700263023376, 0.0014991782372817397, 0.1946621686220169, 0.18547774851322174, 4.5759100914001465, 1.0017813444137573, -0.0008837947389110923, -0.007097452413290739, 0.019592922180891037, 0.08688328415155411, -1.3855289220809937, -3.0516762733459473, 2.086024045944214, -0.12346255034208298, -0.08709108084440231, 0.11807458102703094, 0.8407208919525146, -0.003333432599902153, -0.1789293885231018, -1.9018982648849487, -2.6969797611236572, -0.928422749042511, -0.10019207745790482, 0.432715505361557, 0.2848525047302246, -1.3053334951400757, -0.09687832742929459, 0.8610574007034302, 0.820369303226471, -0.05241591855883598, 2.1789333820343018, 0.005044045392423868, -0.08183267712593079, -0.44671961665153503, 0.8656103014945984, -1.4074918031692505, -0.15348535776138306, -0.19406914710998535, -0.038712020963430405, 0.09906172007322311, -0.49623903632164, 0.429019957780838, 0.549326479434967, -0.03176812827587128, -0.27163439989089966, 0.2601450979709625, -0.666885256767273, -0.15766504406929016, -1.7448256015777588, 0.7019338607788086, -0.48411816358566284, -0.5380083918571472, 0.43332216143608093, 0.334135502576828, -0.42857205867767334, -1.2883780002593994, 0.8457094430923462, -0.33322903513908386, 1.0319817066192627, -0.807594895362854, 0.05731794238090515, -0.25199830532073975, 1.1333450078964233, 0.621521532535553, -0.02350584603846073, -1.3058890104293823, -0.29199472069740295, 0.7623336911201477, 0.024824215099215508, 0.7586743831634521, 0.1589447408914566, -0.218122661113739, -0.8536323308944702, -0.29219892621040344, 1.2831815481185913, -0.6032223701477051, 0.7226150631904602, 0.5933806300163269, -0.10762523859739304, 0.2612973749637604, -0.017641084268689156, 0.769174337387085, -0.6214926242828369, 0.9335213899612427, 1.6395509243011475, -0.9328967332839966, 0.7011752128601074, -2.962899923324585, -0.5965937376022339, 0.6561842560768127, 1.3355315923690796, 0.9332414865493774, -0.0515681654214859, -0.023976942524313927, -0.5880907773971558, -0.8228012323379517, -0.03297368437051773, -0.0153556689620018, 2.725982427597046, -0.0025774880778044462, 0.06925932317972183, 0.20712263882160187, 0.030700916424393654, 0.026257216930389404, 0.17683276534080505, -0.010906022973358631, -0.02702571265399456, -1.3251683712005615, -0.3131701946258545, -1.578287959098816, -0.3448929786682129, 0.4181504547595978, -0.6619532704353333, 0.8001608848571777, -0.01440708339214325, 4.00687837600708, -0.3049777150154114, -0.03593575954437256, 1.7834891080856323, 0.09099540114402771, 1.870516061782837, -0.5672436356544495, 0.3327413499355316, 0.07212653756141663, 0.09276549518108368, 0.8887123465538025, -2.329728126525879, 1.092571496963501, 1.6542431116104126, -0.5953742861747742, -0.07043608278036118, -0.12126002460718155, -0.03583769127726555, 0.5552695989608765, -0.4956606924533844, 0.5118598341941833, 0.581884503364563, -0.08751978725194931, -1.1005630493164062, -0.5142234563827515, 1.265830636024475, -1.4393038749694824, 0.3682343661785126, -0.212296262383461, 0.4913848638534546, -0.34324660897254944, -0.3474231958389282, -0.3401772677898407, -0.718849778175354, 0.3252852261066437, -5.853392124176025, -0.00023326696828007698, 0.023548275232315063, -0.7162749767303467, -1.2642258405685425, -1.0791215896606445, -0.10386109352111816, -0.6683117747306824, -1.0337954759597778, -0.4600207805633545, -0.31196004152297974, -0.010820610448718071, 0.3754657208919525, -0.6149449944496155, -0.1715162694454193, 0.2787910997867584, 0.4329586327075958, -0.3392324447631836, -1.2494938373565674, -2.7437233924865723, -1.27018404006958, -2.848721981048584, 0.03841952607035637, 0.50071781873703, 0.3030529022216797, 0.059082403779029846, -1.5179108381271362, -3.2502691745758057, 0.47911500930786133, -0.46381548047065735, 1.7746132612228394, 0.45458152890205383, -0.06076992675662041, -0.22311004996299744, -0.6346944570541382, -2.0356266498565674, 0.002878461964428425, 1.633463740348816, -0.6987584233283997, 0.3455231785774231, -0.026264991611242294, -0.1883276253938675, -0.9887313842773438, 0.060002345591783524, 0.329036682844162, 0.10385964065790176, -0.16655004024505615, 0.7720517516136169, -0.05099900811910629, 0.09845192730426788, -0.612909197807312, -0.16866770386695862, -0.005329023115336895, -0.07692555338144302, 0.0010070559801533818, -0.9642773866653442, -0.01086729671806097, 0.4222017526626587, 0.14273765683174133, -0.8545120358467102, 0.9891393184661865, -0.8990249037742615, 0.4419545829296112, -0.15586642920970917, -0.352590948343277, -0.36514008045196533, -0.2156096249818802, -0.031934887170791626, -0.3083604872226715, -0.7038163542747498, -0.016254790127277374, -0.4222181439399719, -0.21392133831977844, 0.9919192790985107, 0.35933536291122437, -0.016714157536625862, 0.8300216197967529, -0.04331289231777191, 0.9114086031913757, 0.3669613301753998, -0.21464617550373077, -2.712381362915039, -0.34115099906921387, -0.25141793489456177, -0.043171226978302, -0.13741038739681244, -0.9389156699180603, 1.1533678770065308, -0.9528829455375671, 0.05374162644147873, 0.4001311957836151, 1.1317222118377686, -1.691100001335144, 0.0011273244163021445, -0.014492048881947994, -0.2344435751438141, -0.8867944478988647, -0.1081882044672966, -0.21846827864646912, -0.0909515917301178, -1.00918447971344, -2.358095645904541, 1.743193507194519, 1.2535765171051025, 1.068589687347412, -0.308278888463974, -0.9260654449462891, -0.23153650760650635, 0.2935183048248291, 1.1716970205307007, -0.6136882901191711, 0.46657243371009827, 1.7572846412658691, 0.01569218747317791, -0.32722747325897217, -0.22010383009910583, 0.004344084300100803, -0.5829678177833557, 0.4693469703197479, 0.4639051556587219, -0.21065257489681244, -1.245851993560791, 1.226897954940796, 0.7793846130371094, -0.034101542085409164, 1.098663568496704, -0.2502248287200928, -1.696651577949524, -0.20709189772605896, 0.8814024329185486, 0.04748975485563278, 1.8737963438034058, 0.633269190788269, -0.5921471118927002, 1.6357661485671997, -0.08842067420482635, 0.7719171643257141, -0.27060961723327637, -3.212718963623047, -1.4695518016815186, 0.06245007738471031, -0.8279739618301392, -1.1111072301864624, -0.1295708566904068, -0.5784339308738708, -2.13370418548584, 0.17740005254745483, 0.024035010486841202, -1.0482544898986816, -0.4537760615348816, 0.14263691008090973, 0.07818999886512756, -0.13683652877807617, 0.14783206582069397, 0.8979271650314331, 0.11843453347682953, -0.07081421464681625, 1.2925209999084473, 0.09428493678569794, 0.3991367518901825, -0.04665157198905945, 0.34843575954437256, -0.15896081924438477, -0.0212811678647995, -1.4660483598709106, -1.3138173818588257, 0.9933085441589355, 1.4859954118728638, -0.2412157505750656, -0.09524579346179962, -0.41814136505126953, -0.49141985177993774, -1.6731034517288208, 0.10395431518554688, -0.6595601439476013, 0.13002371788024902, -0.47203126549720764, 0.2582285404205322, 0.04015200585126877, 0.7322711944580078, 0.060402460396289825, 0.9913366436958313, 0.6853808760643005, 0.4352240562438965, 0.26309502124786377, -0.26476648449897766, 0.6356977820396423, -0.14611971378326416, 0.04643585532903671, -0.30560868978500366, 0.31410473585128784, -0.017811238765716553, 0.850973904132843, -0.00016991478332784027, 1.1090041399002075, 0.5551472902297974, 1.1649492979049683, 0.10985306650400162, -0.5279638171195984, 0.07096390426158905, -0.06948183476924896, -0.18088525533676147, 2.8024675846099854, -1.047364592552185, -0.061638690531253815, 0.5786237120628357, -0.02760147862136364, 0.014198924414813519, -0.18469995260238647, -0.30654317140579224, 0.20061570405960083, 0.20794548094272614, -0.03268835321068764, -1.7385135889053345, -0.502334713935852, -0.21412424743175507, -0.4945584535598755, -0.019128208979964256, 0.28589171171188354, -0.10584910213947296, 0.641623318195343, 2.2767577171325684, 0.8228622674942017, -0.5664294958114624, 0.7173445224761963, -1.481711983680725, 0.9855557680130005, 0.47341257333755493, -0.29741549491882324, 0.6098507642745972, -0.5645611882209778, 0.037408001720905304, 0.026443084701895714, -0.23787690699100494, 1.250440239906311, 0.14723002910614014, 0.5560724139213562, -0.18499153852462769, -0.058547765016555786, 0.7336092591285706, 0.26869088411331177, -1.4638643264770508, -1.1817575693130493, -0.49787095189094543, -0.8851576447486877, -1.592029094696045, 0.0011840742081403732, -0.48304933309555054, 0.12783168256282806, 0.24417896568775177, 0.29437434673309326, -0.14034122228622437, -0.040098078548908234, 0.3063720762729645, 0.3139006197452545, -0.8842047452926636, -0.10484752058982849, -0.10412988811731339, -0.08964119106531143, -1.7493960857391357, 0.39167606830596924, -1.5351608991622925, -0.6755452752113342, -0.5488200783729553, 1.5879617929458618, 1.3190107345581055, -0.276713103055954, -0.27256709337234497, -0.34652677178382874, -0.27797114849090576, -0.6065213084220886, -0.40948089957237244, -0.006609943229705095, -0.29514482617378235, -0.3035483658313751, 2.596684455871582, 0.3344932496547699, -0.3592086732387543, 2.9246280193328857, 0.3501608967781067, -0.14900296926498413, 1.6681456565856934, 0.42339107394218445, -0.4065326154232025, 0.39155325293540955, -2.8427765369415283, 0.4446553587913513, -0.06767471134662628, 0.003481527091935277, -0.5621691346168518, 3.1354212760925293, -0.0762743428349495, -0.5564703941345215, -0.5204559564590454, 0.23256637156009674, 0.1959378868341446, 0.26464805006980896, -0.23704320192337036, 0.22573015093803406, 1.4502743482589722, 0.20673567056655884, -0.7727367877960205, 0.20815519988536835, -0.18060080707073212, -0.06687448918819427, -1.5918960571289062, -1.1309741735458374, 0.5647776126861572, -0.06142505258321762, -0.2179378718137741, 0.6706814169883728, -0.37475359439849854, -0.44861525297164917, 0.6503152251243591, 1.5677422285079956, 0.3909074664115906, -0.686558723449707, -0.6415047645568848, -0.051068369299173355, -3.0516154766082764, -0.6518082618713379, -0.16937406361103058, 0.5169294476509094, -0.7838228344917297, 0.6057408452033997, 0.51186203956604, -0.23712009191513062, 0.07492918521165848, 1.7695376873016357, -0.005064132623374462, 0.27561134099960327, -0.12347956746816635, 0.4265545904636383, -0.258694589138031, -2.1599228382110596, 1.870816946029663, -0.2485600709915161, 0.5640774369239807, -0.3470925986766815, -0.49457934498786926, 0.41684022545814514, 0.49355363845825195, 0.011149327270686626, 0.20204941928386688, -0.16598914563655853, -3.0574748516082764, -0.8600882291793823, -1.3079406023025513, 0.013504453003406525, -0.17858096957206726, 0.3783513009548187, 0.0065987505950033665, -1.9453394412994385, 0.8819940686225891, 0.17696192860603333, 0.0003130823897663504, 0.13068927824497223, -0.005259337835013866, 0.3580872118473053, 1.1411123275756836, -0.10493753105401993, 0.18435491621494293, -0.046865805983543396, -0.43261638283729553, 0.28696656227111816, 4.1207075119018555, -0.5855457782745361, 0.013647058978676796, 0.9213593602180481, -0.013699626550078392, 0.43970444798469543, 0.2139425277709961, -0.597586989402771, -0.15667182207107544, -2.2002079486846924, -1.0652512311935425, 0.9376029968261719, 0.8423013091087341, 0.08651714771986008, -1.004137396812439, 0.5766890048980713, -0.5748663544654846, -0.41118064522743225, 0.05305128172039986, 0.31816112995147705, 0.12033125013113022, 0.015457612462341785, -1.1155422925949097, 0.5276861190795898, 0.4133508801460266, -0.24353794753551483, -0.11141122132539749, -0.9301045536994934, 1.0832185745239258, -0.28918471932411194, 0.3489293158054352, -1.0414767265319824, -1.372429370880127, 0.2422775775194168, 0.07523233443498611, 0.11925288289785385, -0.23086436092853546, -0.5661823749542236, -0.1471826583147049, -0.0005609819781966507, -0.12514331936836243, -0.39139068126678467, 0.20483136177062988, 0.09744536876678467, 2.150787591934204, 0.41164714097976685, 0.4259161055088043, 0.0014829197898507118, -1.1595193147659302, -0.5262771844863892, -0.5929738283157349, 1.073334813117981, -1.2065317630767822, -1.819700837135315, 0.6994388103485107, 0.02338423766195774, -0.06489013135433197, 0.12479574233293533, -0.1882983297109604, -0.34058311581611633, 0.04095125570893288, 0.0021662714425474405, -0.06131845712661743, 1.0668907165527344, 0.8565992712974548, 2.09810471534729, -0.45192399621009827, -0.6889428496360779, 0.03644433990120888, 0.8760060667991638, 0.1036878302693367, 0.2245807945728302, -2.241976737976074, 0.30361208319664, 0.1186671182513237, -0.9278350472450256, 0.806665301322937, -0.8126766681671143, 0.4155033826828003, 0.27160000801086426, -0.7354387640953064, -0.42169857025146484, 0.06371752917766571, -0.03634611889719963, 1.0724897384643555, 0.4385761320590973, -0.4808506369590759, 0.3298025131225586, -0.8736165165901184, 0.20037801563739777, -0.49200883507728577, 0.042685411870479584, 1.037154197692871, -0.32683250308036804, 0.4451344311237335, 0.3809342086315155, 0.004641929641366005, 3.896049976348877, 1.7749954462051392, -0.21847109496593475, -0.44421079754829407, 1.1073617935180664, -0.6911975145339966, -0.08853830397129059, -0.010504758916795254, 0.29568785429000854, 0.03659805655479431, -0.10412796586751938, 0.5036875605583191, -0.04517141357064247, 2.276326894760132, 0.3686951696872711, 0.5879935622215271, -2.5041325092315674, -0.14071887731552124, 0.35148826241493225, 0.37525343894958496, -1.021621823310852, -0.13131609559059143, 0.4039059281349182, -1.0934754610061646, -1.251900315284729, -1.5802749395370483, 0.04780106991529465, 2.6995270252227783, 0.010322519578039646, 0.5379939675331116, -0.2004878968000412, -0.20684805512428284, 0.19273202121257782, -3.2314560413360596, 0.5287560224533081, -0.01943640224635601, -0.016054261475801468, -0.9337077736854553, -0.12412365525960922, -0.38397255539894104, 0.3108427822589874, 0.8805662989616394, 1.2543939352035522, 0.2996140122413635, 0.17272290587425232, 0.023243768140673637, 0.8071580529212952, 1.3663763999938965, -1.2525380849838257, 1.089410424232483, 0.003386017167940736, 0.41133424639701843, 1.5490843057632446, -0.838151216506958, 0.007121909409761429, -0.9373041391372681, 0.43247029185295105, -0.7732606530189514, 0.019745970144867897, -0.07903442531824112, 1.1326348781585693, -1.9981451034545898, -0.1168457567691803, -1.764098882675171, 0.6426273584365845, 0.08787211775779724, 0.46271929144859314, 0.7465627789497375, 0.24912790954113007, 0.2623976469039917, -0.11721092462539673, 0.4462706744670868, 0.10381203144788742, -0.34053534269332886, -0.12457511574029922, 0.29820960760116577, -0.07578328996896744, 1.3722327947616577, 1.5439116954803467, 0.04362970590591431, 0.010997996665537357, 2.0729565620422363, 0.029454302042722702, -0.1859014481306076, 0.6836559176445007, 0.2402065098285675, -1.0072091817855835, 0.13999909162521362, 1.5437538623809814, 0.19623661041259766, 0.45503905415534973, 0.9549484252929688, 0.6067493557929993, -0.6705021262168884, 0.14492850005626678, 0.485210120677948, -0.0788445845246315, 0.7104890942573547, 2.4760196208953857, -0.05022665113210678, -0.6031104922294617, -0.5177997350692749, -0.12276467680931091, 1.038783311843872, 1.3275747299194336, 0.6554951667785645, 0.3000692129135132, 0.32430753111839294, -1.0984747409820557, 0.027987370267510414, -0.9735504388809204, 0.6045133471488953, 2.1955971717834473, -0.10030782967805862, -0.010400773957371712, -0.07926676422357559, 0.5512135028839111, -0.03027559071779251, 0.13816708326339722, -0.18884827196598053, -0.26246583461761475, -0.005489891394972801, 0.573963463306427, -0.007988188415765762, -0.8888320326805115, -0.14909248054027557, 0.1722191870212555, -0.6531822681427002, -0.4386083781719208, 1.0999314785003662, -0.13545756042003632, 1.711769700050354, -0.317074179649353, 0.04486241936683655, 0.05832473561167717, -1.003173828125, -0.13514362275600433, 0.08409830927848816, 0.12495416402816772, -1.1278072595596313, 2.4733901023864746, 0.1820344626903534, -0.053792804479599, 0.30867671966552734, 0.3394595980644226, 0.06443612277507782, 2.8480770587921143, -0.29650866985321045, 0.11581233143806458, -0.7939871549606323, 1.2284682989120483, 0.5789441466331482, -0.03139222785830498, 0.3787802755832672, -3.841036558151245, -1.173836588859558, 0.10736951231956482, 0.9955291748046875, -0.14600905776023865, 0.00935004185885191, -0.9283137321472168, 0.4294569492340088, -0.15871010720729828, -1.21097731590271, 0.2325667440891266, 0.6293666958808899, -0.7503973245620728, 0.036831703037023544, 0.7733574509620667, -0.17828607559204102, 0.10293219983577728, 0.49372902512550354, 1.0227125883102417, -2.072289228439331, 0.16906048357486725, -1.3272451162338257, 0.4735410809516907, 1.3284785747528076, 0.06500490009784698, 0.6561669707298279, 0.3266330361366272, -0.21197423338890076, -1.1631128787994385, 0.21786923706531525, -1.9350463151931763, -0.13483870029449463, -0.3586136996746063, -2.2664732933044434, 0.6592323184013367, -0.538915753364563, -1.2361583709716797, 0.07347257435321808, -0.8487675786018372, 0.13861404359340668, 0.795450747013092, 0.3129151463508606, -0.5208522081375122, -0.2552894651889801, 0.9891773462295532, 0.010308035649359226, 0.7475870251655579, 0.5097336173057556, 0.1780865341424942, 2.2248053550720215, 2.865304708480835, -0.5989078879356384, 4.534507751464844, 0.6651868224143982, -0.040165726095438004, 1.9793992042541504, -0.7360924482345581, 0.02857486717402935, -1.5503700971603394, 0.22952522337436676, 0.0014939741231501102, 0.14741460978984833, -0.6943537592887878, -0.16944333910942078, -0.022250255569815636, 0.5609478950500488, -0.4639304578304291, 0.04799377918243408, -0.0006628811825066805, 0.014257272705435753, -0.9715508818626404, -0.020320355892181396, -4.773592472076416, -0.16867640614509583, -0.10963282734155655, 1.7850654125213623, 0.0006335079669952393, -0.019705649465322495, 0.0716627910733223, 0.1765867918729782, -1.3402751684188843, -0.056032534688711166, 0.26671889424324036, -0.18280129134655, -1.0405956506729126, 0.29012924432754517, -0.05811857804656029, 2.0425660610198975, -0.43660426139831543, 2.530024290084839, -0.009951849468052387, 0.09126598387956619, -0.16448388993740082, 0.6841517686843872, 0.4556330144405365, 0.3453788459300995, -0.15843142569065094, -0.08449167758226395, 0.006346622481942177, 0.29109692573547363, 0.023444848135113716, 0.15810726583003998, 0.04137074574828148, 0.7222064137458801, -0.43697595596313477, 1.070097804069519, -0.6278859376907349, -0.07434806227684021, 0.8365225195884705, -0.254599928855896, 0.9148731827735901, 6.3212890625, 0.6732819080352783, 0.1657002866268158, 0.012540115043520927, -0.34210965037345886, 0.49108004570007324, 1.5263912677764893, -0.603309154510498, 0.750907838344574, 0.04458016902208328, 0.8316947817802429, 0.27175989747047424, 0.42952755093574524, -0.5571606159210205, 0.6183158755302429, 0.11708729714155197, -0.7491241097450256, 1.5034422874450684, -0.5005166530609131, -2.6684460639953613, -0.3444950580596924, 0.4983617067337036, 0.6445064544677734, 0.020809167996048927, -0.14083066582679749, -0.7692556977272034, -1.1240137815475464, -0.39576831459999084, -0.35035714507102966, -1.024584174156189, -0.5127567052841187, -0.04828424006700516, 0.0028154735919088125, 0.10961634665727615, -0.085012286901474, 1.0729072093963623, -0.6766301989555359, 0.2722194194793701, -0.4076906740665436, 0.2840670347213745, -0.41125500202178955, 2.6610496044158936, 0.07440990954637527, -1.728659749031067, 0.31892746686935425, -0.7201644778251648, 0.1001092791557312, -1.150693416595459, -0.09589686989784241, -0.318245530128479, 1.2908977270126343, 0.04014017805457115, 0.15157045423984528, -0.32005545496940613, -0.2651955485343933, 0.1475069224834442, 0.05076488107442856, 0.9489885568618774, -0.2188934087753296, 0.45247283577919006, 0.20383387804031372, 0.03586751967668533, 1.3762871026992798, 0.10956210643053055, 0.47533878684043884, 0.3311486542224884, 0.2699611186981201, 1.3334269523620605, -0.26374128460884094, -0.1957974135875702, 0.1744435429573059, -0.008597685024142265, 0.2329532504081726, 0.361587792634964, -0.0736955776810646, 0.021339276805520058, -0.7363901734352112, -1.1372307538986206, -1.4625039100646973, 0.0035074949264526367, -1.6350445747375488, -0.9278454780578613, -0.056301623582839966, -0.7741807103157043, 0.6813045144081116, 0.3209577798843384, -2.575666904449463, 0.1252915859222412, -0.568215012550354, 1.4383554458618164, 1.8162044286727905, -0.5411011576652527, 0.04267842322587967, 1.4432874917984009, 0.3817945718765259, 1.4793965816497803, -0.10919546335935593, 0.6234901547431946, 0.20849169790744781, -1.269381046295166, -0.06552165746688843, -0.4870636463165283, 1.1638458967208862, 1.5878291130065918, 0.15076130628585815, -0.17321650683879852, -0.2648114264011383, -0.8310871124267578, -0.08028789609670639, 0.4917429983615875, 0.5461054444313049, 0.7063826322555542, 0.05489218235015869, 0.12116187065839767, 0.6648483276367188, 0.7642165422439575, -2.547685146331787, -0.06955385953187943, 0.12988373637199402, -1.806789755821228, -0.32665929198265076, 0.1189320906996727, -0.0031186959240585566, -0.45984065532684326, -0.33132556080818176, -0.0011541673447936773, -1.7294526100158691, -0.09513864666223526, 1.1188560724258423, 0.08695909380912781, -0.30807653069496155, -0.004785424098372459, -0.386772096157074, 0.03940282389521599, -0.10892746597528458, 0.008484093472361565, 1.2743196487426758, 0.06958119571208954, 0.39286598563194275, -0.2546057105064392, 0.32577961683273315, 0.6754224896430969, 0.9214614033699036, -0.37475553154945374, 0.8212620615959167, 0.4899960458278656, 0.04839090630412102, 1.0329172611236572, 0.260737806558609, -3.409839153289795, 0.10566399991512299, -0.13923315703868866, 0.0059336512349545956, 0.4903072118759155, 0.1973683089017868, -0.1447521448135376, 0.01729283295571804, -0.04188532382249832, 0.5579386949539185, -0.9283684492111206, 1.2169761657714844, 0.37324970960617065, 0.6499082446098328, -0.2910183072090149, -0.5398843884468079, 2.060091495513916, 0.17558546364307404, -0.008968505077064037, 0.2979390621185303, -0.10910400748252869, 0.14997032284736633, 0.19580769538879395, 0.044157739728689194, -0.3338770866394043, 0.014689580537378788, -0.13893240690231323, -0.13208423554897308, -0.13319753110408783, -1.5300273895263672, 0.1514093428850174, 0.6002391576766968, 0.19932673871517181, -1.6755845546722412, 0.09273213893175125, -0.180877685546875, 0.6751664280891418, 0.347946435213089, -1.9399381875991821, -2.0120067596435547, 0.4107537865638733, 1.9318801164627075, 2.330949544906616, 0.2152262181043625, 0.2174670696258545, -0.05160132050514221, -0.36212536692619324, 0.520140528678894, 0.853536069393158, -0.7831494808197021, 0.4212930500507355, 0.41784122586250305, 0.026804227381944656, 0.029283059760928154, 0.12290836870670319, 0.5113635063171387, -0.17633101344108582, -0.22900916635990143, 1.0954532623291016, -1.3161976337432861, 0.26296889781951904, -0.09112083166837692, 0.7495179176330566, -0.38204848766326904, -0.14766459167003632, 1.3245335817337036, -0.24881817400455475, 0.05172104761004448, -0.09061958640813828, -1.3267474174499512, -1.1559723615646362, 5.182531356811523, 0.04610694572329521, -0.42617067694664, 0.34281083941459656, -1.427675485610962, -0.07812222093343735, -1.9150888919830322, -0.6210335493087769, -0.26236918568611145, -0.012405231595039368, -0.8188262581825256, 0.13143931329250336, -0.17183096706867218, 0.5579573512077332, 0.10710459202528, 0.5198591947555542, 0.026155199855566025, -0.037658825516700745, -0.1161474883556366, 0.04868209734559059, -0.5207811594009399, -0.24589549005031586, -0.38014060258865356, -0.4633198082447052, -0.00025517583708278835, 0.674452006816864, 0.09746899455785751, -0.9811632633209229, -1.2267341613769531, 1.8131707906723022, 0.7285406589508057, -0.867799699306488, 1.9333469867706299, 0.012013772502541542, 0.7414856553077698, -1.198112964630127, 0.4097229838371277, -0.01918524131178856, 1.2849998474121094, -0.4677935838699341, -0.042876068502664566, 0.1479608714580536, -0.012627706862986088, -0.3001410961151123, -1.8166290521621704, -1.7153006792068481, -0.026095813140273094, 0.303282231092453, -1.8877267837524414, 0.5070478916168213, 0.0043545071966946125, -0.3405000567436218, -0.08804454654455185, -1.0139249563217163, 0.19584530591964722, 0.05641544982790947, -0.1261935532093048, 0.030976343899965286, -0.4781792163848877, 0.02043052390217781, 0.16322259604930878, 0.1377277970314026, -0.035600047558546066, 2.240060329437256, 0.08275274932384491, -0.3139643669128418, -0.29017511010169983, -0.0950436145067215, 0.19444195926189423, 0.4010879099369049, -0.2465924322605133, -0.007334805093705654, -0.09124323725700378, -2.621919631958008, -3.0220682621002197, 0.3020186126232147, -0.014114382676780224, -2.578692674636841, -0.06269428879022598, 0.563500702381134, -0.3970625102519989, 0.08986712247133255, 0.22476565837860107, -0.1285625398159027, -0.8172832727432251, 2.041931629180908, -1.3294936418533325, -0.053912948817014694, 0.1762627810239792, 0.0018194700824096799, 0.6045074462890625, 0.39702779054641724, 0.16924460232257843, -0.6757261753082275, -0.5383620262145996, 0.5880018472671509, 0.011884059756994247, -0.5439193844795227, 0.7711091041564941, 0.05146191641688347, 0.0004995677154511213, -1.3252086639404297, -0.8250826001167297, -0.049798235297203064, -0.008978454396128654, -0.7578176856040955, 0.19588622450828552, -0.30090951919555664, -0.7170409560203552, -0.10586470365524292, -0.02571232244372368, -0.2703007757663727, -0.05361570045351982, 0.11154346913099289, 0.10950130969285965, 0.2857648432254791, -0.6900332570075989, 0.6975769400596619, -1.9699956178665161, 1.3267321586608887, 0.6971414685249329, 0.3223170340061188, 0.05103910714387894, -0.06658279150724411, 0.31590524315834045, 0.32273516058921814, 0.11361569166183472, 0.21998919546604156, -0.017756983637809753, -0.0959978699684143, -0.3545209765434265, -0.3394373953342438, -1.1435691118240356, -1.4003618955612183, 1.5134717226028442, 0.6007320284843445, 0.7545651197433472, 0.5696427822113037, -2.262270450592041, 0.14022982120513916, 0.14943553507328033, 0.18227821588516235, 0.4418654143810272, 0.08332782238721848, -0.04502895101904869, -1.0008721351623535, 0.6980531215667725, 0.9628670811653137, 0.5413103103637695, -0.4023001492023468, -1.7438907623291016, 0.427603155374527, -0.40596553683280945, -1.5648540258407593, 0.06683273613452911, 0.13292114436626434, 0.28609439730644226, 0.45367631316185, 0.9420220851898193, 1.727760910987854, -0.5542603135108948, -0.8325329422950745, -0.9032644033432007, 1.6573835611343384, -0.3197762370109558, -0.06298618018627167, 0.24687233567237854, 0.21898514032363892, -0.10036774724721909, 0.4835510849952698, -0.07439093291759491, -0.17792057991027832, 0.2653754651546478, -0.7913827300071716, 0.5937604308128357, -2.6439647674560547, 0.07201480865478516, -0.2854875326156616, -0.4344644248485565, 1.2817621231079102, 0.033567603677511215, -0.23875385522842407, 0.688352108001709, 0.245167538523674, 0.14025695621967316, 0.15927433967590332, -0.06200293451547623, 3.0854856967926025, -1.3198493719100952, -0.019963771104812622, 0.5349876284599304, -1.5007902383804321, 0.00022031976550351828, 1.1465156078338623, 0.17349034547805786, 0.4862242341041565, -0.7860881686210632, 0.4252401292324066, 0.02695787511765957, -1.3737459182739258, 0.1323605477809906, 1.9498271942138672, 0.38612693548202515, 0.09556467086076736, 0.03340279310941696, 0.08159218728542328, -1.719136118888855, 0.2467096745967865, 0.2902171015739441, 0.0381646454334259, -0.1262073814868927, 0.06967059522867203, 0.6885671019554138, 0.7028387784957886, 0.20202311873435974, -0.3006718158721924, -0.5039644241333008, 0.02316123992204666, -0.04946502670645714, 1.297816276550293, -0.40013429522514343, 0.42022788524627686, 1.5683958530426025, -3.3646886348724365, 0.05190177634358406, 1.4865342378616333, -0.31951186060905457, -0.33290618658065796, -3.183846950531006, -0.021962514147162437, 0.01010120753198862, -0.060552503913640976, -0.6674820780754089, -0.5054570436477661, -0.5282151103019714, -0.1307677924633026, 0.7964891791343689, 0.3652074337005615, 0.22411775588989258, 0.3030437231063843, -0.6131500601768494, -0.12641900777816772, -1.7499209642410278, 5.7429585456848145, -0.0029747767839580774, -0.7774547338485718, 0.012139681726694107, 0.5208696722984314, -1.6671761274337769, 0.8390061259269714, -0.08688512444496155, 0.043573569506406784, 0.26264622807502747, 0.45168089866638184, -0.49661412835121155, -0.604199230670929, 1.300980806350708, -1.0453956127166748, -0.731340229511261, 1.3150140047073364, 0.2963741421699524, -0.5778811573982239, 1.8102989196777344, -0.5235680341720581, -0.2595624029636383, -0.34677207469940186, -1.3152836561203003, 0.5073215961456299, 0.07252579182386398, 0.2977786958217621, -0.34430521726608276, -0.44054433703422546, 1.5938471555709839, -0.010730760172009468, 1.9349489212036133, -0.5356673002243042, -0.5834048986434937, -0.6410055756568909, 0.34241941571235657, 0.6485522389411926, -0.23233796656131744, -1.2587828636169434, 0.011272044852375984, -0.004114413168281317, 0.8082488179206848, -0.24429570138454437, 1.025771141052246, -0.3137359619140625, 0.16398391127586365, -0.25671273469924927, 0.19955086708068848, 0.17208431661128998, 0.1759125143289566, 0.33821266889572144, -0.05416509881615639, 0.13339386880397797, 0.6536791324615479, 0.030575405806303024, -0.28101029992103577, -0.05355439335107803, -0.4876740276813507, 0.288529634475708, 0.10173878073692322, 0.7049360275268555, 0.015381558798253536, 0.20967531204223633, 0.4890631139278412, 2.2118327617645264, -0.244767963886261, -2.554123878479004, -0.7462670207023621, -0.6846374869346619, 2.698146343231201, 0.6117876172065735, 0.17847363650798798, 0.5767055749893188, 0.2703763544559479, -0.7190930247306824, 0.07895006239414215, 0.5868809819221497, -0.06461232155561447, -0.3701713979244232, 1.2257546186447144, -2.086642265319824, -0.9701677560806274, 0.04448740929365158, -0.023951582610607147, 2.180727243423462, -1.0023781061172485, 0.04444476217031479, 0.6953963041305542, -0.6681470274925232, 0.29965388774871826, 2.0708742141723633, 0.01566036231815815, -0.24566859006881714, 0.10365451127290726, -0.0666058212518692, -0.3633108139038086, -1.5781338214874268, -1.0619463920593262, 0.11933168768882751, -3.4844579696655273, 0.2486676722764969, -0.39500564336776733, -0.12082070857286453, 0.6983591914176941, -0.4756767451763153, -0.6222429871559143, 1.0401585102081299, 0.6083289384841919, 0.06125302240252495, -0.24210602045059204, -0.08821732550859451, -0.05875281244516373, -0.5941386222839355, -0.5667611360549927, 0.19012697041034698, -0.694124698638916, 0.132610484957695, 0.06315328180789948, -0.566740870475769, -1.631679892539978, -0.48858585953712463, 0.8238701820373535, 1.0109527111053467, -0.31745317578315735, 0.05086476728320122, 0.011288139969110489, 0.1628023236989975, 0.6341704726219177, -0.3605446517467499, 0.8400576710700989, 0.46308550238609314, -0.11576374620199203, -0.7459934949874878, -0.5371242761611938, 0.06258318573236465, 0.6750534176826477, -0.0555296428501606, -0.1175091415643692, -0.09722736477851868, -1.4545209407806396, 0.25572890043258667, 0.1313457190990448, -0.023451246321201324, -0.1876072883605957, -2.2581679821014404, 0.028026290237903595, 0.13775958120822906, 0.2743503153324127, 0.012108461000025272, -0.4118119776248932, -0.5897228121757507, 0.8797133564949036, 0.55403733253479, 0.5975440144538879, -3.3652613162994385, 2.3611183166503906, -0.062478482723236084, 0.04322047904133797, -0.7013130187988281, -1.1597048044204712, -0.10284442454576492, -0.28377288579940796, -0.18631340563297272, 0.0656074732542038, 1.0542528629302979, -0.6003773212432861, 0.004207503516227007, 2.366339921951294, 0.19488857686519623, 0.16524171829223633, -0.7688082456588745, 0.0816844254732132, 0.09499332308769226, 0.18741567432880402, 0.4226383864879608, -0.5385412573814392, 0.7795310020446777, 0.6868599653244019, -3.304499626159668, 0.21247471868991852, -1.4705301523208618, 0.013842238113284111, -0.005044476129114628, -1.1464687585830688, 0.361870676279068, -0.7778832912445068, 0.03262905403971672, -0.5686350464820862, 0.5352563858032227, -0.4349658787250519, 0.9730239510536194, -1.1880587339401245, -0.05233088508248329, 0.12447929382324219, -0.005274808965623379, 0.029344778507947922, 0.11374460905790329, -0.08055607229471207, 0.0023455587215721607, 0.058879584074020386, -3.7021989822387695, 0.13992750644683838, -0.5147413015365601, 3.0099446773529053, -0.5930014848709106, -0.6341105699539185, 0.27127471566200256, 1.1520966291427612, 0.4343359172344208, 0.06225353479385376, 0.39073583483695984, -0.006273524835705757, 1.1054848432540894, -0.6250969171524048, -0.059984494000673294, -0.02142445556819439, 0.01741647906601429, -0.7287085652351379, 0.34164899587631226, 0.3888925015926361, -0.15986526012420654, -0.08792383223772049, -0.775838315486908, -0.5781177878379822, -0.035707972943782806, 3.814258575439453, -0.4265686571598053, -2.176901340484619, 0.2066907286643982, 0.6413812041282654, -0.05834716930985451, -0.22117085754871368, -0.12207956612110138, 0.17772026360034943, -0.049789637327194214, 0.9018311500549316, -0.025446098297834396, -2.4413070678710938, -0.18610988557338715, -0.8026069402694702, 0.3226549029350281, 1.2303913831710815, -1.6620904207229614, -1.094873070716858, -0.2724563181400299, -2.754380464553833, 1.3562369346618652, 0.13403993844985962, -0.3819080591201782, -1.1448098421096802, -0.4614220857620239, -1.1665527820587158, 0.4865020215511322, 1.0128612518310547, 0.6943825483322144, -0.3866504728794098, 1.0446463823318481, 0.1146746352314949, -0.5347243547439575, 0.04529033973813057, 1.6418813467025757, 0.9310325384140015, 1.511624813079834, 0.20389209687709808, -0.030028870329260826, -0.33398616313934326, 0.29854103922843933, 0.34264352917671204, 0.9399088621139526, -2.3835971355438232, 1.4634894132614136, -0.2272215336561203, -1.4501886367797852, 0.027954809367656708, -0.04160508140921593, -0.8597897887229919, 0.20939511060714722, 0.4796389639377594, 0.5830541849136353, -0.0653737336397171, 0.7287641763687134, -0.7601020932197571, -0.23524898290634155, 0.3611614406108856, -0.37693077325820923, 0.9477686882019043, 0.5334747433662415, -0.854812502861023, 0.02495826967060566, -0.31421586871147156, -0.6983217000961304, -0.25106534361839294, 0.4534778594970703, -0.6397145390510559, 1.2357453107833862, -0.19066467881202698, 0.7834659218788147, -0.04858633503317833, -0.0012065739138051867, 0.8656463623046875, 0.5599181652069092, 0.4203404486179352, 3.065156936645508, -0.06860217452049255, -0.7430028319358826, -0.9592859745025635, 0.15956008434295654, -5.229381084442139, 0.21196968853473663, 1.4934985637664795, -0.8293485641479492, 1.6241719722747803, 0.5164605379104614, -0.0017486715223640203, 0.06002497300505638, -0.1598815619945526, 1.4106370210647583, 0.27220094203948975, 0.8671612739562988, 0.01861126720905304, -0.14727671444416046, 0.07153026759624481, 0.31093278527259827, -0.07369156926870346, 0.7882091999053955, 0.42698267102241516, 0.3297121822834015, -0.21219795942306519, -0.44866636395454407, -1.2332594394683838, 1.784830093383789, 0.5950301289558411, 1.64622962474823, 0.4277985990047455, 1.1087298393249512, -0.00042347409180365503, 3.402761220932007, 0.008138916455209255, 0.20562750101089478, 0.4568192958831787, -0.2726069688796997, 0.1540968418121338, 0.16720139980316162, 0.6851659417152405, -0.3692498505115509, -0.23673246800899506, 0.4941830039024353, 0.07583066821098328, 1.05048406124115, 0.17185555398464203, -2.370439052581787, -1.1679421663284302, -0.39886850118637085, 0.0670349970459938, 0.913505494594574, -2.7338476181030273, 0.4546803832054138, -0.40749889612197876, -0.07298760116100311, -0.0453089214861393, -0.26885542273521423, 0.7549930214881897, -0.8346832990646362, -1.4169056415557861, 0.01407143659889698, -0.3010643720626831, -1.6722917556762695, -0.060914285480976105, -0.6337953209877014, -1.2061526775360107, -3.4197235107421875, -1.134238839149475, 0.16362862288951874, -0.564000129699707, -3.220646381378174, 0.19031372666358948, 0.05023694410920143, -0.6074370741844177, -0.26505038142204285, -0.13154925405979156, 0.012018267065286636, -0.684195876121521, -1.7754579782485962, 0.08391635119915009, -0.18376903235912323, -0.1126064732670784, 0.6952268481254578, -0.8877093195915222, -0.029545346274971962, 0.47054898738861084, -0.03720731660723686, 0.0010711285285651684, 1.3061413764953613, 0.4181094765663147, 0.5382686853408813, 0.15069998800754547, -0.089466392993927, 1.0330699682235718, 0.2114183008670807, 0.012634671293199062, -0.015593467280268669, -0.6667740941047668, -0.009095240384340286, 0.045033156871795654, 0.8657529950141907, -0.598334014415741, -1.0228253602981567, -0.05188453942537308, -1.3104503154754639, -0.025520745664834976, 0.33434462547302246, -1.7588343620300293, -0.028549162670969963, -0.4783242344856262, -0.010980943217873573, -0.28491491079330444, -0.9669709801673889, 0.5060445666313171, -0.06320497393608093, 4.7705535888671875, -0.022283906117081642, 1.0388411283493042, 0.3130301535129547, -0.49930018186569214, 1.9924670457839966, -0.8205997943878174, -1.4164237976074219, -0.8634845614433289, 0.19978733360767365, 0.3663819432258606, -0.013512992300093174, 0.21363478899002075, 0.46539849042892456, 1.1183301210403442, -0.4222327470779419, -0.01934756711125374, -0.24736274778842926, 0.3514584004878998, -0.18333610892295837, 0.45923927426338196, -0.01688091643154621, 0.042358268052339554, -0.8619928956031799, 0.12540623545646667, 0.4402608573436737, 0.05490156635642052, -0.38765573501586914, 0.23638102412223816, -0.03323265165090561, -0.7925708889961243, 0.09719990938901901, -0.1159989982843399, -0.46197131276130676, 1.0965771675109863, -0.06264833360910416, 0.17598547041416168, -0.049514222890138626, 0.6138991713523865, 0.19160600006580353, 0.7583903074264526, 0.7440276741981506, -0.0017503103008493781, -0.0032035172916948795, -0.637718915939331, 0.4954786002635956, -0.7163395285606384, -0.22894880175590515, -0.019550036638975143, 1.393502116203308, -0.1766616404056549, -0.013314936310052872, 0.07368462532758713, -0.028209445998072624, 0.6033409833908081, -1.597806453704834, 0.9827609062194824, -0.728123664855957, -0.0035050828009843826, -0.01689351722598076, -0.7839398980140686, 3.887047052383423, 0.31387993693351746, 0.06547581404447556, -0.344143271446228, 0.7199053764343262, -0.08931372314691544, 0.07560490071773529, -0.7336450815200806, 0.4668810963630676, 0.005592211615294218, -0.409286230802536, 2.0682175159454346, -0.01910415291786194, -1.3027842044830322, 0.10793452709913254, -0.9496108293533325, 0.20974880456924438, -0.08293598890304565, 0.8989145159721375, 0.20018255710601807, -0.7767471671104431, 0.22809116542339325, 0.17807164788246155, -1.187194585800171, -0.10083331912755966, 0.12683650851249695, -0.43491870164871216, 0.577953577041626, 0.18599660694599152, 0.12930676341056824, 0.11134583503007889, -2.621082305908203, -0.007501895073801279, 0.4084993004798889, -0.03872419148683548, 0.35474032163619995, 1.8723810911178589, 0.08651678264141083, 0.03952617943286896, -1.2370622158050537, 2.6823911666870117, -0.2592279613018036, 0.029044033959507942, 0.16969554126262665, -0.2271685153245926, 0.5059542655944824, -1.13852059841156, -0.005812978371977806, 0.28210127353668213, 0.06905930489301682, -0.04996732249855995, -0.06966348737478256, -0.15739890933036804, -0.05931477248668671, -1.8027263879776, -0.07687779515981674, -0.5586543679237366, -0.36515888571739197, -1.7003239393234253, 0.827653169631958, 0.03698268160223961, -0.4365330636501312, -0.9117768406867981, 0.682969331741333, 0.00025931521668098867, -1.3009014129638672, -0.2421656996011734, 1.4409016370773315, -0.41354748606681824, -1.2718026638031006, -0.19771869480609894, -0.0637078732252121, -0.13677248358726501, 0.4304758310317993, -0.03051634132862091, 0.8356897830963135, 0.0012854806846007705, -1.3419675827026367, -0.22873274981975555, -0.9682357311248779, 0.5460293292999268, -1.1603466272354126, 1.3535845279693604, 1.4989513158798218, 0.17389339208602905, -0.3715662658214569, -2.871472120285034, 0.028644656762480736, 1.11156165599823, -1.887799859046936, -0.11033986508846283, 0.9733066558837891, 0.0928594097495079, -0.358302503824234, -0.3148735463619232, -0.36556312441825867, 0.21175797283649445, -0.9997636079788208, 0.15890952944755554, 0.3651438057422638, -0.3779342770576477, -0.09080684930086136, -1.4340766668319702, -1.1398411989212036, -0.5241743922233582, 0.16356351971626282, 0.150384321808815, -0.09606003016233444, -0.0980478823184967, 0.6632345914840698, 1.115706443786621, -0.042216379195451736, 1.0407387018203735, -0.04047538340091705, 0.6387918591499329, 1.0302523374557495, 0.6126755475997925, 0.04219428822398186, 0.032938145101070404, -0.10952016711235046, 0.6904134750366211, 0.6992943286895752, -1.567064642906189, 1.1701899766921997, 0.3967973291873932, -0.290149986743927, 0.8266308307647705, 1.6818537712097168, 0.19492115080356598, -0.6486507654190063, 0.6064800024032593, 0.10426024347543716, -0.22588418424129486, 1.7376773357391357, -0.939565122127533, -0.08974673599004745, 1.4178303480148315, -1.3854458332061768, 0.9830645322799683, 0.3832375109195709, -1.1113510131835938, -0.5663993954658508, -0.5880535840988159, -0.3131707012653351, -0.30469194054603577, -0.8237671256065369, -1.016706109046936, 0.1289418786764145, 2.125887155532837, -0.04887019842863083, 0.371295303106308, 0.08542169630527496, 1.2251087427139282, -0.9192833304405212, -1.3445088863372803, 0.05245465785264969, -0.0007399010355584323, 1.2284694910049438, 0.07500304281711578, 0.0012019480345770717, -0.06901220977306366, 0.09887971729040146, 1.7170120477676392, -1.4456543922424316, 1.8626976013183594, -0.2546384632587433, -0.06820840388536453, -0.07678181678056717, 1.0423237085342407, -0.22819600999355316, -2.6810555458068848, -0.21023109555244446, 1.0100528001785278, 0.660683810710907, 0.07853466272354126, 0.06937863677740097, -0.16033677756786346, -0.3626982569694519, -0.2912207245826721, -0.8220105171203613, 0.461424857378006, 0.6224668622016907, -0.12555401027202606, 0.6306819319725037, 0.1007927730679512, 0.423414945602417, 0.2165256142616272, -0.7442455887794495, -0.004203601740300655, -0.4584987163543701, 0.05954727903008461, 1.1352168321609497, 0.3282580077648163, -0.8512229919433594, 0.831913411617279, -0.20613564550876617, 0.17124676704406738, 0.17387908697128296, 0.002850706223398447, 0.050216905772686005, 0.016116784885525703, 1.4266244173049927, 0.5674185156822205, 0.3159404397010803, 0.01649743691086769, 0.06787440925836563, 0.004768907558172941, -0.10092387348413467, -0.684808611869812, -0.17634260654449463, 0.046155158430337906, -0.014394924975931644, -0.35019931197166443, -0.5915982127189636, -2.9436120986938477, 0.6034738421440125, -0.9679323434829712, -0.4679326117038727, 1.1433619260787964, -0.08372168242931366, 0.02244114875793457, 0.1594027876853943, 0.03928491473197937, 1.737859845161438, -1.2754861116409302, -0.0495077446103096, -0.1547776162624359, -2.8288352489471436, -0.2952750623226166, -0.5729555487632751, 0.512077271938324, 1.3642287254333496, -0.7499487996101379, 1.0650955438613892, 2.6663806438446045, 0.09560521692037582, 0.16648103296756744, 3.6776177883148193, 2.3386738300323486, 0.5247690677642822, -2.1044914722442627, -0.5698188543319702, 0.7727299928665161, 0.01378221157938242, -2.7216176986694336, -0.25458842515945435, -1.320037841796875, -0.1670379489660263, -0.5409716367721558, 1.5700393915176392, -0.43423253297805786, 0.24768325686454773, 1.130637288093567, -0.12876224517822266, 0.10571226477622986, -0.46725431084632874, 0.0627417266368866, 0.8368755578994751, 0.1658654659986496, -0.03200286254286766, -0.3831771910190582, -0.49876055121421814, 0.018467899411916733, 0.35789379477500916, -0.18111832439899445, -0.1953383833169937, 0.1790618896484375, 0.7116657495498657, 0.11925545334815979, -1.1522659063339233, -2.255864381790161, -0.23735490441322327, -0.11512349545955658, 0.0632421001791954, 0.9044851660728455, 0.18962964415550232, -0.1983201950788498, 0.543939471244812, 0.26871246099472046, -2.7199227809906006, 0.1235310509800911, 0.6704332828521729, -0.07989193499088287, 1.7456672191619873, 3.454474449157715, -0.4033917784690857, 0.2594881057739258, -2.0941994190216064, -0.12738478183746338, 0.09711074084043503, -0.07695623487234116, -0.005380317568778992, 0.3088301718235016, -0.11061916500329971, -0.0955246314406395, -0.6286850571632385, 0.07401393353939056, -2.5653727054595947, 0.396218866109848, 0.14056876301765442, -0.99444180727005, 0.10567224770784378, 0.13778889179229736, 1.0744563341140747, -0.7705626487731934, -0.33281001448631287, -0.06873969733715057, -0.015612181276082993, 0.34599968791007996, -0.12117063254117966, 0.011331194080412388, 0.02009028010070324, 0.08221906423568726, -0.007552136201411486, -0.8557298183441162, -0.6916589736938477, 0.4048628807067871, 2.22589111328125, -0.2429182231426239, -2.614407777786255, 0.13920576870441437, 0.23312507569789886, -1.3431706428527832, 0.27137142419815063, -0.1628255695104599, -0.0073276604525744915, 0.27631479501724243, -1.0406981706619263, 0.4410260021686554, -0.4413280487060547, -2.1131234169006348, -0.10589263588190079, 0.11176177859306335, -0.9089647531509399, 1.154569387435913, 0.24980022013187408, -1.2024260759353638, -0.04179645702242851, 0.5316426753997803, 0.02567168138921261, 0.8677710294723511, -0.023838339373469353, -0.42264747619628906, -0.006291033234447241, 0.3763117492198944, 0.06025642156600952, -0.7946280241012573, 0.07892454415559769, 0.4228461980819702, -2.022963285446167, 1.810012698173523, -1.5832583904266357, -1.6305421590805054, 0.2105843722820282, -0.16539792716503143, -0.5482726693153381, -0.8244172930717468, -0.040827009826898575, -0.008020380511879921, -0.18404754996299744, -0.2405351847410202, 1.5969984531402588, 0.9321277141571045, 0.994772732257843, -0.17942999303340912, 0.17867325246334076, 0.1158645898103714, 0.02060454897582531, -0.08345034718513489, -0.010904665105044842, 0.5694739818572998, 3.1031911373138428, -0.17187777161598206, 0.985765278339386, 0.21110832691192627, -0.5746798515319824, -0.49939027428627014, -0.29596608877182007, -1.8109608888626099, -0.34053391218185425, 1.7880514860153198, -1.3253328800201416, 0.07354225963354111, 1.1551991701126099, -1.3885692358016968, 0.7864929437637329, 0.12851011753082275, 0.6442567706108093, 0.43717119097709656, 1.0773768424987793, -0.595228910446167, -1.4001600742340088, 0.7899284958839417, -0.6717645525932312, 0.633739709854126, -4.990509033203125, -0.2674495577812195, -2.3557052612304688, -0.8590891361236572, 0.27391231060028076, 0.013296322897076607, -0.5643752813339233, -0.3656827211380005, 0.004939092323184013, 0.4780948758125305, 1.213623046875, -0.21479342877864838, -0.7144683003425598, 0.9892441034317017, -0.26139765977859497, -0.3139120936393738, 0.2408474236726761, 0.1608857363462448, -0.35818272829055786, 0.538582980632782, -0.12125696241855621, 0.2754574120044708, 0.36835846304893494, 0.11628004163503647, -0.728732705116272, 0.03707456216216087, -1.3173717260360718, 1.9579707384109497, -0.23144283890724182, -2.139415740966797, 0.8912830948829651, 0.10039032250642776, 0.6285955309867859, 0.5104787945747375, 0.3082387447357178, -0.6820247769355774, 0.0027111379895359278, -0.45052775740623474, 0.10075273364782333, 0.7025097608566284, -0.30968666076660156, 0.665049135684967, 0.7554938197135925, -1.2840787172317505, -0.6655633449554443, -0.1775302141904831, -0.8087438941001892, -0.3488689064979553, 0.2696647644042969, 1.7953040599822998, -0.593616783618927, -1.6416361331939697, -0.06533734500408173, -0.9197908639907837, 0.12030629813671112, -0.012913801707327366, -1.1069905757904053, 0.31086933612823486, -0.4876093566417694, -0.8033261895179749, 2.4404289722442627, -0.22150208055973053, -0.8086212277412415, -0.4185614585876465, 0.3560183346271515, -0.9868594408035278, 0.2708472013473511, 0.20650967955589294, 0.610362708568573, 1.3647798299789429, -0.7716633677482605, -0.5340526103973389, 0.28346556425094604, 3.27950119972229, 0.6956204771995544, -0.12279431521892548, 0.2581179440021515, -2.43078875541687, 1.5209195613861084, 1.5832349061965942, 0.1588330715894699, 0.9395939707756042, 0.0573045015335083, -0.24564337730407715, -0.546492338180542, -0.17059218883514404, 0.08460890501737595, 0.8809810280799866, 0.23655107617378235, 0.15477237105369568, -1.0058871507644653, 0.7288508415222168, -1.400476336479187, -0.28866180777549744, -0.003408054355531931, -4.44840669631958, -0.509131908416748, -0.1270265430212021, -1.2953298091888428, 0.15743406116962433, -0.9124792814254761, 0.13170528411865234, 0.4133017063140869, -0.009631434455513954, -0.7669178247451782, 0.12311086803674698, -0.2146804928779602, 0.149173304438591, 0.8568773865699768, 0.9011765718460083, 0.15701620280742645, -0.4892716705799103, 0.2867061197757721, -0.6994354128837585, 0.017834613099694252, -1.4410873651504517, -0.0008555541862733662, 3.5631191730499268, -0.0649612620472908, 0.13316960632801056, 1.489538311958313, 0.12617401778697968, 1.7254973649978638, 1.6932717561721802, -1.0004340410232544, -0.5036914944648743, -0.08424311131238937, -0.05531484633684158, -0.8818941712379456, -0.583795428276062, -0.5798273086547852, -0.2700551748275757, 0.1217103824019432, 0.005607552360743284, -1.893630862236023, 0.01704421453177929, -2.1089115142822266, 0.10377004742622375, 0.8242698907852173, -0.8314053416252136, 1.7095357179641724, -0.33963945508003235, -1.564732313156128, 0.996736466884613, 1.2095744609832764, 2.3763928413391113, 0.00276566157117486, 0.7637069821357727, 0.04136237874627113, -0.9471410512924194, -0.15960916876792908, 0.036880966275930405, 0.05115026235580444, 0.8810592889785767, -0.6068429350852966, 0.837143063545227, -0.25261589884757996, 1.0853102207183838, 4.421562671661377, 5.54888916015625, -0.7268826961517334, -0.830840528011322, 0.41342756152153015, -0.41456907987594604, 0.2461429089307785, -0.05081683397293091, 0.036279719322919846, -0.2512999176979065, -0.05029205605387688, -0.8698377013206482, -0.4657479524612427, -0.07702863961458206, -0.1350332796573639, 0.011682789772748947, 0.03394431248307228, -0.02737630531191826, -0.008369311690330505, -0.2973587214946747, -0.2313978224992752, -0.23862488567829132, 0.06762511283159256, -1.533634901046753, -0.6794998049736023, -0.762144923210144, -0.7225685715675354, -0.03281557187438011, 0.10632443428039551, -1.3799140453338623, 0.133781298995018, -0.04227951914072037, 0.17744846642017365, 0.4235394299030304, -0.08565249294042587, 2.275926113128662, 0.16792169213294983, -1.716799259185791, 0.19761838018894196, -0.11885815858840942, 0.0263774823397398, 0.6939054131507874, 0.4093506932258606, -0.9395906925201416, 3.381779670715332, 2.0652644634246826, 0.3262740671634674, -0.08834372460842133, -0.4822219908237457, -0.34148791432380676, 0.4241567850112915, -0.13060665130615234, 0.8212219476699829, 2.4195966720581055, -0.35195910930633545, 0.07970516383647919, -0.059843145310878754, 0.0028453601989895105, 1.0449328422546387, 1.3495707511901855, -0.1280873715877533, 0.6908630728721619, 0.46544063091278076, -0.5419430732727051, -0.7872322797775269, -2.391270399093628, -1.0035511255264282, -2.885545015335083, -0.7087268233299255, -3.6546876430511475, -0.08036784082651138, -1.4382789134979248, -0.33959269523620605, 1.1544848680496216, -0.3219064176082611, 0.2536686360836029, 0.10567086935043335, 1.335440993309021, -0.6047645807266235, 0.5611987113952637, 0.011265615001320839, 4.576152324676514, -1.7058459520339966, 0.08411574363708496, 1.0956752300262451, -0.16825568675994873, 0.4315510392189026, 1.3339807987213135, 2.540440320968628, 0.2548573613166809, -0.6600880026817322, 0.0043162074871361256, -0.09106595814228058, 0.4791520833969116, -0.11543644964694977, 0.07638034969568253, 0.5528638958930969, 0.043992601335048676, -0.678135335445404, 0.24194906651973724, 3.8456032276153564, -0.007524695713073015, -0.14680670201778412, -0.09164376556873322, -2.3848605155944824, -0.8202242851257324, -0.9159355163574219, 0.2296316921710968, -0.8759102821350098, 0.4250201880931854, 0.8368463516235352, -0.8583201766014099, 0.39468351006507874, 0.16048270463943481, -1.4656281471252441, -0.11634330451488495, -0.27644163370132446, -0.0673866719007492, 0.029069053009152412, 0.0533284917473793, -1.0829108953475952, -0.4712453782558441, -0.680096447467804, 0.9463866949081421, -0.8307327032089233, 0.4225788712501526, -0.572111964225769, 0.7560240030288696, -0.00019158174109179527, 6.591300964355469, 0.49285614490509033, -0.03724740073084831, -0.48027166724205017, 0.43707090616226196, 2.5199058055877686, 0.2759406268596649, 0.1496768444776535, 0.6787365078926086, 1.7283663749694824, -0.1877494603395462, -0.0016117463819682598, 0.2509288787841797, 0.6699329614639282, 0.044852983206510544, -0.847665011882782, -0.33235499262809753, -0.07034973800182343, 0.7666515111923218, -1.8660356998443604, -0.023550525307655334, -0.003547444473952055, -0.32697200775146484, 1.2323321104049683, -1.085475206375122, -0.06449860334396362, -1.6330846548080444, -0.0912565216422081, -1.7510042190551758, -0.2562394440174103, 0.05298037827014923, -0.9985741972923279, 0.6802912950515747, -0.651854395866394, 0.20365434885025024, -0.39643383026123047, -0.10932273417711258, 0.5757882595062256, -1.6351803541183472, 0.1128702238202095, 0.27663734555244446, -0.0270890761166811, 0.25608018040657043, -0.28462547063827515, -0.2249635010957718, -1.6086502075195312, -0.02660864032804966, -0.20355212688446045, -0.534645676612854, 0.15395915508270264, -0.024970127269625664, -0.8753068447113037, 1.9990251064300537, 0.8351907730102539, 0.04276706278324127, 1.6594609022140503, -0.32354751229286194, 2.3185980319976807, 0.5927799940109253, -0.11774999648332596, 0.8792968392372131, -0.04934636503458023, -0.04733112081885338, -0.6109395027160645, 1.537199854850769, 0.1303974986076355, -1.133063793182373, -0.7213103175163269, 1.1788588762283325, 0.04593725875020027, -0.8972100019454956, -0.7876113057136536, -3.2538137435913086, -2.5290732383728027, 0.09391984343528748, 0.22767242789268494, -1.0687686204910278, 0.16828447580337524, -0.01878722757101059, 0.009307806380093098, 0.605592668056488, -1.9797178506851196, -0.8165236115455627, 1.9549801349639893, -0.383821964263916, 0.3845926821231842, 0.044292300939559937, -0.1944047510623932, 0.17279337346553802, -0.9300337433815002, 0.12582600116729736, 0.38803768157958984, 0.08019717037677765, -0.09637120366096497, -0.5495226383209229, 2.7396697998046875, 0.08358842134475708, -0.20692172646522522, 0.2362486720085144, 1.1074289083480835, 0.6303839087486267, 0.035419657826423645, -0.4463610053062439, -0.027454407885670662, -0.24230414628982544, -0.5556765198707581, -1.1408817768096924, 0.13254427909851074, 0.047405753284692764, -0.011265010572969913, -1.9987326860427856, -0.030291715636849403, -0.0729493796825409, 0.10031983256340027, 0.19567222893238068, 2.0364017486572266, -0.42670267820358276, -0.1152334064245224, 0.09989875555038452, -0.21768656373023987, 1.097046971321106, 0.015602795407176018, -0.14942272007465363, -1.6165975332260132, 3.6742966175079346, 1.6303350925445557, -0.06396546959877014, -1.74811851978302, -1.242806077003479, -0.11257510632276535, 0.028285156935453415, 0.19030430912971497, 0.01598750613629818, -0.26216647028923035, 0.05130738392472267, -0.13448090851306915], [-0.05322064831852913, -1.2262167930603027, 0.18918967247009277, 1.0725430250167847, 0.32693466544151306, -3.4019339084625244, 0.027798566967248917, 1.122968316078186, -0.3604050278663635, 0.9459988474845886, -1.579488754272461, 0.1992253065109253, 0.4576762914657593, 0.10784713178873062, -0.17368900775909424, 0.5300318598747253, 0.7088724374771118, -0.2959592044353485, -0.07426903396844864, -0.02343754656612873, -1.2077364921569824, 0.2124452441930771, 0.009068016894161701, -0.43669790029525757, 0.03710804134607315, 0.08668740838766098, -0.6898627281188965, -0.12886571884155273, -2.5391340255737305, 0.026989029720425606, 0.22753594815731049, 0.28472182154655457, 0.055560141801834106, -0.0064526027999818325, -0.08323672413825989, -0.7998530268669128, 1.2072179317474365, -0.4990757405757904, 0.10773933678865433, -0.7726629376411438, 1.3519389629364014, 0.6938017010688782, 0.3545636534690857, -0.3450952470302582, 0.04336755722761154, 0.282554030418396, -0.06666889041662216, 0.17318430542945862, -0.2746574878692627, -0.3127983510494232, 3.178507089614868, 0.10876812040805817, -0.03604168817400932, 0.14073903858661652, 0.048278167843818665, -0.6241249442100525, 0.30431613326072693, -0.002637029392644763, 0.09394901990890503, 0.03639078512787819, 0.1734081357717514, -0.18170985579490662, -0.5328086018562317, 0.8417958617210388, 3.3037381172180176, -0.2936784029006958, -0.06642631441354752, 0.08266234397888184, -0.03850064054131508, -0.2645190954208374, 0.3500465750694275, -2.9205853939056396, -1.4571460485458374, 2.6460838317871094, -0.7597188353538513, 0.21496005356311798, 0.31224507093429565, 2.519796848297119, 0.025980837643146515, 0.28396928310394287, 1.2292568683624268, -0.5969105362892151, 0.1305675208568573, 0.07034170627593994, -0.3253868520259857, -0.019687095656991005, -0.10741191357374191, -0.8968451619148254, 1.691490888595581, -0.9216706156730652, 1.6139177083969116, -0.06575275957584381, 0.11532486230134964, 0.7108572125434875, 0.2526742219924927, -1.6804155111312866, -0.11358370631933212, -0.6489063501358032, 0.5392621755599976, -0.10645509511232376, -0.0433901846408844, 1.2028456926345825, 0.7360860109329224, -0.047805704176425934, 0.007950849831104279, 0.04873088002204895, 0.5005031824111938, -0.28990671038627625, -0.2610678970813751, 0.462172269821167, 1.2431607246398926, 0.025508856400847435, 0.4667814373970032, 0.11292316019535065, 0.10366711020469666, -0.2661173939704895, -1.3242655992507935, 1.6564122438430786, -1.0893323421478271, 0.8085317015647888, -0.06952647864818573, -0.39648881554603577, 0.15745075047016144, 0.42316296696662903, -0.06506875902414322, -0.22332297265529633, 0.1451897919178009, -0.03884613886475563, -1.6480793952941895, 0.1919192224740982, -0.49873924255371094, 3.6380136013031006, -0.020359070971608162, -1.2251720428466797, -0.11400462687015533, -0.5453770756721497, -0.009048770181834698, 0.126038059592247, 1.3366265296936035, 0.05231118202209473, -0.825892448425293, -0.09800132364034653, -0.04379931464791298, 0.06856739521026611, -0.4009026288986206, 1.1775870323181152, 0.6800519227981567, 3.055180311203003, -0.004896162077784538, 0.23717001080513, 0.2487153261899948, -0.5743426084518433, -0.06087205559015274, 0.4703221917152405, -0.31253567337989807, 1.5701560974121094, -0.8122501969337463, -0.031014334410429, -0.5281133055686951, -1.2532044649124146, 0.9316639304161072, -0.04428279772400856, 0.009254605509340763, 1.8014370203018188, 0.4946359395980835, -0.03644265979528427, -0.010611442849040031, -1.5767356157302856, 0.3866170346736908, 0.008484031073749065, 1.081418514251709, -0.8372374773025513, -0.34518781304359436, 2.3909506797790527, -0.2797788381576538, 0.16113051772117615, -0.1598464548587799, -0.42185550928115845, 0.8877997994422913, 0.828315794467926, -0.9468339085578918, -0.02550402469933033, -0.001713427365757525, 0.1934177726507187, -0.5435856580734253, 0.6858440637588501, 0.5805315971374512, -0.014765078201889992, 0.37692099809646606, -3.8063533306121826, -0.028712758794426918, -1.0421513319015503, 0.24607624113559723, 0.05247948691248894, -0.008130007423460484, -0.0311224777251482, 1.7317081689834595, 0.2983523905277252, -3.106173515319824, -0.9597030282020569, 0.15762794017791748, -0.17347052693367004, 1.3428651094436646, 0.06432335078716278, 1.296783208847046, 0.1677165925502777, -0.8047623634338379, 0.9784153699874878, 0.39780524373054504, -0.3797305226325989, -0.03608180955052376, -0.3275718092918396, -0.22958633303642273, 0.035002678632736206, 0.18765991926193237, 2.0919721126556396, 0.6494560837745667, 0.3630208671092987, 0.004246596246957779, -0.17050404846668243, -0.11650100350379944, -0.5522562265396118, -0.22979983687400818, -2.7296669483184814, -0.189309760928154, -0.11856168508529663, -0.5839834809303284, 1.397883415222168, 1.5336977243423462, 0.0801108106970787, -0.3267765939235687, 0.7450026273727417, 0.24605156481266022, 0.15585941076278687, 0.6731032729148865, -0.10137716680765152, -0.31530988216400146, 0.3143574595451355, 0.3344649374485016, -0.010716727003455162, 0.19092829525470734, 0.9429547190666199, 1.0003437995910645, 0.9880641102790833, 0.6690462231636047, -0.06437116116285324, 0.9892231225967407, 2.523362874984741, 0.1799871176481247, 0.2305992990732193, -0.02621774561703205, -1.6137903928756714, -2.0357108116149902, 0.17708832025527954, 0.5027393102645874, -0.2234639674425125, -0.5773380994796753, -0.11394382268190384, 0.015623594634234905, -0.7103028297424316, -0.333191454410553, -2.6287612915039062, 0.5412485003471375, 0.929597020149231, -0.4025268256664276, -1.2696386575698853, -4.590285778045654, 0.5596588253974915, 1.3483645915985107, -0.8480715751647949, -0.0014192884555086493, -1.5862940549850464, -0.2308361977338791, 0.709132969379425, -0.23053856194019318, -0.2529771625995636, -0.2749057412147522, -0.18528729677200317, -1.4773430824279785, 1.493623971939087, 0.12915025651454926, -0.5947456359863281, 0.18800418078899384, 0.08656596392393112, -0.08399413526058197, -0.09387689083814621, -0.5348861813545227, 1.5113884210586548, 0.09761377424001694, 0.7274974584579468, 0.060702379792928696, -0.21307091414928436, 0.4327397346496582, 0.8309205174446106, 0.11909976601600647, 0.46668559312820435, -0.38948437571525574, 0.7925361394882202, -0.3861512541770935, 0.012870554812252522, -0.743384063243866, -0.019756251946091652, -0.024431798607110977, 0.4904392957687378, -2.310244083404541, -0.751602828502655, -0.17863032221794128, -1.036486268043518, -0.5669479966163635, -0.9629067182540894, 0.0381019189953804, 1.6933369636535645, 0.12427125871181488, 0.44766080379486084, -0.41288504004478455, 0.31375429034233093, 1.9277448654174805, -0.05231710150837898, -0.7192921042442322, 0.044111113995313644, 0.12917201220989227, 0.025306792929768562, 1.3864070177078247, -0.4089641869068146, -0.033111732453107834, -0.3936585485935211, 1.8090782165527344, -1.250425100326538, -0.07177286595106125, 1.1141968965530396, -0.8811559081077576, 0.32632240653038025, 0.019979704171419144, 0.008566171862185001, -0.2508297562599182, -1.4921473264694214, -0.632215678691864, 1.4197651147842407, 0.13911640644073486, -0.8958146572113037, -1.0783050060272217, -0.36189544200897217, 0.5617814064025879, -2.4018709659576416, -0.7944024801254272, 0.5460379719734192, 0.21466529369354248, -0.04934267699718475, 0.18881410360336304, 0.06826989352703094, 2.2137608528137207, -0.9865643978118896, -1.2471535205841064, -1.2616848945617676, 0.3852066695690155, 0.6681914329528809, 0.21202914416790009, -0.6159987449645996, 4.74289083480835, 1.122883677482605, 0.22263827919960022, -0.00388721632771194, 0.2419501394033432, -0.4534502625465393, -0.8527379035949707, -0.46664899587631226, 0.8583131432533264, -0.8966213464736938, -0.3402141332626343, 0.2574857473373413, 0.012079603970050812, 0.3568616807460785, 0.005847160238772631, 1.3890020847320557, -0.8506662845611572, 0.12172000855207443, -0.1964719146490097, 0.02488706074655056, -1.2984626293182373, 0.5204654335975647, -0.5551597476005554, -1.4964872598648071, -0.0025664286222308874, -0.6031712889671326, -1.3310710191726685, -0.09407533705234528, 0.08332031965255737, 0.1365024894475937, -0.5360727310180664, 0.6847726702690125, 0.789906919002533, -0.10304832458496094, -0.1945267617702484, 0.12012125551700592, 0.9574422240257263, -0.017995910719037056, -0.11025634407997131, -0.14447875320911407, -0.16448943316936493, 0.00021162573830224574, 0.23101060092449188, 0.9266594052314758, 0.08929909765720367, 1.0294963121414185, -0.01452640537172556, 0.5807391405105591, -2.097158432006836, -0.08965416252613068, 0.1161935031414032, -1.1572974920272827, 0.3929644227027893, 0.6545706987380981, 0.7103607058525085, -1.0521807670593262, 0.007847068831324577, 0.8491905927658081, -0.34110328555107117, -1.8317419290542603, -0.035252247005701065, -0.8529981374740601, -1.1915624141693115, 0.7463327050209045, -0.053891438990831375, 1.3523311614990234, -1.0403721332550049, 0.4279751479625702, -0.04245927557349205, 1.4132376909255981, -0.24520793557167053, -0.45510607957839966, 0.4382154047489166, -0.058722738176584244, -0.9173285365104675, 2.781047821044922, -0.2986357510089874, 0.5405332446098328, 0.4161439836025238, -0.0854540541768074, 0.05843183398246765, 2.090036153793335, 1.6972532272338867, -0.5403444170951843, 0.6708986759185791, 0.11516933143138885, -2.013892889022827, -1.2184473276138306, 2.222365140914917, 0.18613304197788239, -1.4731916189193726, 0.008991852402687073, -1.2052017450332642, -1.1266518831253052, 0.00021166693477425724, -0.18304139375686646, -1.166780710220337, -0.411575585603714, -0.08293644338846207, -0.39584049582481384, -0.23592408001422882, 0.012781293131411076, -0.09118462353944778, -0.23961101472377777, 0.02318200096487999, 0.1729743480682373, 0.2786705493927002, 0.5188114047050476, 2.8668887615203857, -0.04104010388255119, 2.001617193222046, -0.4071952998638153, 0.9592742919921875, 0.2558389902114868, 0.11985699832439423, 2.038661241531372, 0.1122850701212883, -0.5880119204521179, 1.5075523853302002, -1.847922682762146, -0.0344366617500782, 2.126354455947876, 0.4346820116043091, 0.36509233713150024, 0.049057502299547195, 0.005352344363927841, -1.1341633796691895, 5.558220386505127, -0.18221895396709442, 0.48221197724342346, 0.20199595391750336, 0.001245587831363082, 0.14893120527267456, 0.2710508406162262, 0.8451657295227051, 1.3105098009109497, -0.10490504652261734, 0.18224632740020752, -1.4136241674423218, 0.2748766839504242, -1.6756113767623901, -1.6928385496139526, -4.100367069244385, -1.5262117385864258, 0.06215982139110565, -0.017710458487272263, -0.9814261198043823, 0.009672852233052254, 0.01808377541601658, -0.9653253555297852, -0.26629143953323364, 0.3866885006427765, -0.8118542432785034, 0.11645917594432831, 0.1347886174917221, -0.8710740208625793, 1.1881924867630005, -0.1313742697238922, -0.02318558469414711, -0.11978606879711151, 1.249308705329895, 0.605941653251648, 0.6627980470657349, 1.2528733015060425, 0.46907752752304077, -0.10250881314277649, -0.14164260029792786, -0.15857480466365814, 1.1580512523651123, -0.10156945139169693, 0.16074174642562866, -0.16956806182861328, -0.730051577091217, -0.5863339900970459, 0.28794246912002563, 0.06480053067207336, 0.9683632254600525, 0.5981325507164001, -1.0708167552947998, 0.7095059156417847, 0.8811848759651184, 0.7244951128959656, -0.07194589823484421, 0.0027548056095838547, 2.702850341796875, -0.06498144567012787, -0.3038763999938965, -0.030660254880785942, 0.033093709498643875, -0.04410142824053764, -0.21692827343940735, 0.014142470434308052, 0.09693643450737, -0.1268530637025833, -0.4782540798187256, -0.03878110274672508, -1.2936636209487915, 0.5859053730964661, -0.6492748260498047, -0.6196467280387878, 0.1409614235162735, 0.10592952370643616, -0.3394615054130554, 0.13735991716384888, 0.11981990933418274, 1.6695337295532227, -1.2350424528121948, 2.002647876739502, -0.39133384823799133, -0.42186954617500305, -1.9525684118270874, -0.7412223815917969, 0.6730255484580994, 0.9191360473632812, 0.3615504801273346, -0.7149924039840698, 0.3755394518375397, -0.13411563634872437, -1.9838778972625732, -0.07102030515670776, 2.5637564659118652, 0.09072238951921463, 2.7688355445861816, -0.472576379776001, -0.06554640829563141, 0.055193010717630386, -0.1982881873846054, 2.665147542953491, 0.08967892825603485, -0.6713212132453918, 0.5704987049102783, -0.011823914013803005, 0.3898032605648041, -0.08746176213026047, 0.11574919521808624, 0.23217739164829254, -0.004245863761752844, 0.054539360105991364, -1.406273365020752, -0.2060042917728424, -1.2931082248687744, 1.4341837167739868, 0.21150603890419006, -0.1692121922969818, -0.2711256742477417, -1.8370745182037354, 0.08458001166582108, 0.8240758776664734, -1.294007420539856, -0.46793293952941895, -0.9484764933586121, -0.0006550914840772748, -3.657442092895508, -0.1488758623600006, 0.2226041555404663, 0.23908494412899017, -0.34376755356788635, 0.7741633057594299, -0.11875548958778381, 0.6468195915222168, 0.6156116724014282, -0.18092341721057892, -0.0023438234347850084, 0.30197784304618835, 0.09488680213689804, 1.0629295110702515, 0.06769121438264847, -0.14554110169410706, 0.2416943460702896, 0.279056191444397, -0.3262154757976532, 0.32879889011383057, 0.33544647693634033, 0.11485499143600464, 0.01154092513024807, -0.40035179257392883, 0.04054635018110275, 0.3659244477748871, -0.44047194719314575, 1.0678373575210571, 0.019856087863445282, -1.706670880317688, -1.0688334703445435, 0.08608031272888184, 0.4016348123550415, 0.9063696265220642, -0.04812599718570709, -4.311702251434326, 0.6791788339614868, 0.37953805923461914, -0.6681582927703857, 0.043453264981508255, 1.390586495399475, 0.7081730365753174, 0.08718188107013702, 0.5749782919883728, -0.05597912520170212, 0.2179412543773651, 0.006126489955931902, 0.08738818019628525, 1.2132692337036133, -1.653922438621521, -0.22465386986732483, -0.17635099589824677, -0.1586839258670807, 0.02492189221084118, -0.09760543704032898, 0.09309355169534683, -0.1567588746547699, -0.6635987162590027, 0.6450690031051636, 0.10376419872045517, 1.2929062843322754, -0.21586686372756958, 1.2065609693527222, -2.0833630561828613, -1.829283595085144, -4.418783664703369, 1.5723295211791992, -0.2315721958875656, -0.03708761930465698, -0.03923482447862625, 0.23359112441539764, -0.17729173600673676, -0.16883543133735657, -1.5033785104751587, -0.23062346875667572, -0.11126036196947098, -0.06222441792488098, -0.7177633047103882, -1.1531037092208862, 0.3887304961681366, -2.8058228492736816, -0.21335871517658234, 0.25309425592422485, -1.4311593770980835, 0.0919845849275589, 0.7993305325508118, 0.29005876183509827, -0.11454413831233978, -2.7276694774627686, -0.6153241991996765, 0.13920444250106812, -0.9378758668899536, -0.10550783574581146, -0.08877859264612198, -0.058487314730882645, 2.559523820877075, 0.14086732268333435, -0.14963562786579132, 1.7024989128112793, -1.0569871664047241, -1.5769888162612915, -1.5095406770706177, 1.0560925006866455, -0.015477304346859455, -0.24069640040397644, -1.0488237142562866, -0.12037325650453568, 0.19074921309947968, 0.2552071511745453, 0.19984333217144012, 0.33335766196250916, -0.20567749440670013, 3.265061616897583, -0.7740914225578308, 0.9613150954246521, -2.096126079559326, 0.1130647137761116, -0.20194017887115479, 0.29934215545654297, 0.7656716704368591, -0.6023909449577332, -0.3456093370914459, -0.4925050437450409, 0.004887765739113092, 1.1858389377593994, -0.33915114402770996, 0.07619141787290573, 0.12922683358192444, 0.10679478198289871, -1.0462032556533813, 0.0005443923291750252, -0.39485862851142883, -1.4700391292572021, -0.5475781559944153, -0.013073202222585678, 0.3960469663143158, -0.00018686965631786734, -0.23718151450157166, -1.106544852256775, 0.6075427532196045, 0.6183202862739563, -0.9255538582801819, -2.1626272201538086, -0.3369360864162445, -0.8373785614967346, 0.3629498779773712, 0.30395036935806274, 0.037757810205221176, -0.058325063437223434, -0.09058357030153275, 0.0750681459903717, -0.037985559552907944, 0.6015597581863403, -0.42193731665611267, -0.5954347252845764, -1.7672802209854126, -0.06103530898690224, -0.11309418827295303, 8.225343663070817e-06, 0.4957473874092102, -0.005473934579640627, 1.4391589164733887, 0.14523139595985413, 0.0577002614736557, -0.006921744905412197, 0.7137800455093384, 2.3524255752563477, 0.26446619629859924, 0.426982581615448, 1.2490915060043335, 0.005434325896203518, -0.3336242139339447, 0.2025987207889557, -0.5773932337760925, 0.3731279969215393, -0.02402803860604763, -0.38126838207244873, -0.672530472278595, -1.2225661277770996, 1.1373311281204224, 0.32969242334365845, -0.06510058045387268, 0.132801815867424, 0.13593818247318268, -0.001272699679248035, 0.4094732105731964, 0.2534686326980591, -0.1424165815114975, 0.3541324734687805, -0.13289138674736023, -0.7937019467353821, 0.05553405359387398, -0.04496939480304718, -0.9343053698539734, -0.6446079015731812, -0.3641255497932434, 1.4436928033828735, 3.257692813873291, 0.22890090942382812, -2.2738535404205322, 1.1359907388687134, -0.10755657404661179, -0.9983516335487366, 0.03967170789837837, 0.040215592831373215, -0.13441799581050873, 0.023981286212801933, 0.037432488054037094, 0.9926771521568298, 0.2984166145324707, -0.4293803870677948, 1.439679503440857, 0.4381978511810303, -1.297279715538025, 1.3011571168899536, 1.0007784366607666, 0.024357736110687256, 0.27888938784599304, 0.39303022623062134, -0.5228264331817627, -0.5627182722091675, -0.0688997283577919, 0.25967350602149963, 0.13393943011760712, -1.982898235321045, 1.2130472660064697, -0.4862808287143707, -1.8182281255722046, 0.09136392921209335, -0.34359219670295715, 1.8632129430770874, 0.22231896221637726, -0.07847390323877335, -2.526581048965454, -0.892245352268219, -0.3196931779384613, 1.6119589805603027, 0.7266764044761658, -0.004321269225329161, -0.048436056822538376, 0.24974779784679413, 0.3958566188812256, 0.3456478416919708, 0.12086931616067886, 0.13801652193069458, 2.0705270767211914, 0.01049082912504673, 0.15613318979740143, 0.0005530162598006427, -1.6678391695022583, -0.01862427219748497, 0.770792543888092, 1.493330717086792, -0.06786009669303894, 0.42266926169395447, -0.0003536803706083447, 0.799323558807373, 0.10713986307382584, 0.08431261032819748, -1.0285406112670898, -0.3628598749637604, 0.24871426820755005, -1.3937960863113403, 0.005012142471969128, -0.24400626122951508, -0.4774158298969269, -0.032820090651512146, 0.13525548577308655, 0.05626546964049339, -0.06378605961799622, -0.8349103927612305, -0.1949232965707779, 0.13300995528697968, 0.042017143219709396, -0.08849085867404938, 0.1918621063232422, -0.08673134446144104, 0.5129141807556152, 0.32510149478912354, 0.6916772723197937, 0.5034418702125549, -1.2920998334884644, 0.03611836954951286, 0.5590817928314209, -0.3389207124710083, 0.8142635822296143, 0.9165329933166504, -0.466511994600296, -0.10016972571611404, 0.41559863090515137, 0.8045880198478699, 0.2729637622833252, -0.46012216806411743, 0.2797452509403229, -0.8278570175170898, 0.5811664462089539, -0.11573152244091034, 0.8801950812339783, 0.3238731920719147, -0.07377206534147263, 0.18319346010684967, 0.6136069297790527, -0.6523632407188416, 0.10338406264781952, 0.5340097546577454, -0.5149897336959839, 0.39882805943489075, 2.212764024734497, 0.1168505996465683, 1.2331511974334717, -0.0765724778175354, 1.3592052459716797, 0.08708387613296509, -0.7549785375595093, -0.9089084267616272, -0.07014396786689758, -0.20017202198505402, -0.09053942561149597, -0.292979896068573, 0.3395892083644867, 0.22420360147953033, -1.1225868463516235, -0.42123711109161377, -0.6950653195381165, -0.10978017002344131, 1.3109158277511597, -0.10873231291770935, -0.13312479853630066, 0.4426170289516449, -0.3030003309249878, -2.075003147125244, 0.07952029258012772, -0.06238050386309624, 2.3017818927764893, -0.07066449522972107, -0.013414614833891392, -0.07496201992034912, -10.184710502624512, 0.08651866763830185, -0.4019264280796051, -0.08665233850479126, -0.0831209123134613, -0.09518979489803314, -0.4157509505748749, -0.7948715090751648, -0.3871976137161255, -0.1973484456539154, -2.672062873840332, -0.5499562621116638, 0.07619835436344147, 3.917928457260132, 0.9816802144050598, -0.8387092351913452, -0.8535951972007751, 0.8370832800865173, -0.7215014100074768, -0.6986244916915894, -0.045109886676073074, -0.13322603702545166, 2.0342633724212646, -0.28456369042396545, -0.581327497959137, 0.570837140083313, -0.011076334863901138, 0.5930128693580627, -0.1094580814242363, -0.27260422706604004, -0.11482641100883484, -1.2650940418243408, 0.16611644625663757, -0.2076077163219452, 0.389712393283844, 0.5421237349510193, -0.008856261149048805, 0.4407362639904022, 0.7446660995483398, -2.032285451889038, 0.12535542249679565, -0.009465302340686321, 0.3716380000114441, 0.2273196429014206, -0.17160113155841827, -0.0699014663696289, -1.7449110746383667, -0.050272416323423386, 0.3602229952812195, 2.5810060501098633, 0.34478721022605896, -1.0548551082611084, -0.9664356112480164, 3.9136266708374023, -0.5457697510719299, -0.8053943514823914, -0.9826881885528564, -0.3754272162914276, -0.0635528489947319, -0.11555790156126022, -2.0486931800842285, 0.05105039104819298, 0.69831383228302, -0.032684311270713806, 0.8613075017929077, -2.3894667625427246, 1.2621928453445435, 0.7592496871948242, 0.6640099287033081, -0.8091689348220825, -0.44005200266838074, 1.4876652956008911, 1.4858365058898926, 0.4596291482448578, 1.827507495880127, -0.37683841586112976, 0.0007583669503219426, 0.0767674446105957, -0.12837469577789307, 0.05110538750886917, -0.44868430495262146, -0.2549351155757904, 0.768726646900177, 1.2354618310928345, 4.17427921295166, -0.5699360370635986, -0.3300030827522278, 0.787811815738678, 3.3147659301757812, 0.2822767496109009, -1.358969807624817, 1.1460280418395996, 0.019918490201234818, -0.6545584201812744, 0.5715910792350769, 0.06607218086719513, 0.03607252985239029, 0.6390486359596252, -1.0686521530151367, -0.4316323697566986, 0.0009023575112223625, 0.08564100414514542, 0.7753320932388306, -0.09125729650259018, 0.08751825988292694, 0.6124181151390076, 0.41125908493995667, 0.3996124267578125, -1.3221756219863892, 1.0024724006652832, 2.0431110858917236, -0.036826182156801224, 0.1835113763809204, 0.4038081467151642, -1.3650398254394531, -1.316934585571289, 0.07592879980802536, -1.7267986536026, -0.09856666624546051, 0.07160375267267227, 0.02044837921857834, 0.28540492057800293, -0.037319667637348175, -0.19286924600601196, 0.27789536118507385, 0.011298106051981449, 1.6877694129943848, -0.7444916367530823, 0.0034595613833516836, -2.551173210144043, 0.010833518579602242, -0.5767412185668945, 0.03254004195332527, 4.143586158752441, 0.09580020606517792, -0.2001812756061554, 0.21092288196086884, -0.3528035283088684, -0.03337422013282776, 1.8831026554107666, 0.02411196380853653, 0.16186688840389252, 0.3538513779640198, -0.08626453578472137, 1.041162133216858, 0.053539808839559555, -3.1118733882904053, 0.002713915193453431, -0.9406476616859436, 2.1880061626434326, -0.011940930038690567, -0.2900395393371582, 0.0640982836484909, 4.190983772277832, -0.07389186322689056, 0.6240841746330261, 0.1842036098241806, -0.5695033669471741, -1.155434012413025, -0.7510147094726562, -0.3339981734752655, 0.6775875091552734, 2.332834482192993, -1.132683515548706, 0.15300071239471436, 0.45259910821914673, 0.23752541840076447, 0.11432687193155289, -0.7763078212738037, -0.03525499254465103, -1.017188310623169, -0.3074839115142822, 0.09351024031639099, -0.23471781611442566, -0.9185651540756226, -1.1728460788726807, 0.030095277354121208, 0.08453931659460068, 0.13596074283123016, 0.06925088167190552, -0.6375062465667725, 0.5203419327735901, -0.693230152130127, 0.17632919549942017, -0.02422313392162323, -0.0795101597905159, 0.7996045351028442, -0.02509511262178421, -0.3385871350765228, -0.4143095314502716, 0.08357292413711548, 0.7867433428764343, 0.7096492052078247, -0.4183178246021271, 1.2033588886260986, -0.0015872202347964048, 0.16705015301704407, 0.4023626744747162, 0.7884831428527832, -0.056486908346414566, -0.1924581378698349, 0.2999245822429657, -0.11147455871105194, -0.28872352838516235, -0.2855941653251648, -0.21809525787830353, 0.032568857073783875, 0.6179115176200867, 2.8907148838043213, -0.9731704592704773, -0.02568657696247101, 0.15340034663677216, -0.3562853932380676, -0.28641438484191895, 1.3064723014831543, 2.3164894580841064, 0.020526425912976265, -0.9689093232154846, -1.426090955734253, 0.16022446751594543, 0.09844253212213516, 0.34568095207214355, 0.09788206964731216, 0.07762552797794342, 0.2397753745317459, -0.1081671193242073, -0.02758042700588703, 1.2103475332260132, -0.21972379088401794, -1.3551579713821411, 0.31149086356163025, 0.16791993379592896, 0.31474974751472473, 0.19721023738384247, 0.22103744745254517, 0.42443445324897766, -0.006926860194653273, -0.09025464206933975, 0.3155266344547272, 0.11479680240154266, -0.9411522746086121, 0.0037994191516190767, -0.20069622993469238, 0.6747236847877502, 0.059597454965114594, 2.172065019607544, 0.010176431387662888, -1.3921056985855103, -1.1299859285354614, 0.4319310784339905, 0.4378283619880676, 1.448304533958435, -0.07132947444915771, 0.5031142830848694, -1.2584832906723022, 0.15826573967933655, -0.02234748750925064, -0.14273881912231445, 0.43946072459220886, 0.22869358956813812, 0.08341141790151596, 0.09107177704572678, 0.19915343821048737, 0.03493725135922432, -0.08429787307977676, -1.178875207901001, -0.06706275790929794, -0.14705270528793335, -1.3050814867019653, 0.4338596761226654, 0.7436023354530334, 0.44924867153167725, 0.7817916870117188, -0.015572860836982727, 0.1022697240114212, -0.943058967590332, -1.4004379510879517, -0.11223083734512329, -0.3792129456996918, -0.8803191781044006, -0.1931583136320114, 0.36886081099510193, 0.5844763517379761, 0.12914559245109558, 1.1621403694152832, 0.6179133653640747, -3.7430546283721924, 0.11380291730165482, 1.480716347694397, -0.3084820508956909, 0.1465311199426651, 0.3363085687160492, -1.5683966875076294, 0.5108211636543274, -0.9017982482910156, 0.1051659882068634, -1.039571762084961, -0.18478545546531677, 0.001655631116591394, 0.5515896677970886, 1.1637578010559082, 0.7727360725402832, -0.08116047084331512, -0.6430538296699524, -0.7063394784927368, 0.7239100933074951, -0.7749894857406616, -0.08399131149053574, -1.546738862991333, -0.18537071347236633, 0.039795272052288055, 1.3193451166152954, -0.7580760717391968, -0.03135235235095024, 0.22186897695064545, 2.5383851528167725, 1.6989418268203735, -1.2005434036254883, 0.5645873546600342, 0.38876524567604065, -0.6048691868782043, -0.347121924161911, 0.09273693710565567, 0.2667061686515808, 0.10722040385007858, -0.09714172780513763, 0.30431801080703735, 1.5320091247558594, -0.3814775347709656, 0.5810831189155579, 0.07709093391895294, -2.0960733890533447, -0.10335673391819, 2.670583963394165, 0.6342942714691162, 0.8089173436164856, -0.0076452684588730335, -0.02035963162779808, 1.2638747692108154, 1.1697826385498047, 0.9195704460144043, 0.05670660361647606, 0.3262644410133362, -0.16076618432998657, 0.08126167207956314, -4.5075154304504395, -0.23547790944576263, -0.027604050934314728, 0.3388851284980774, -0.5496572852134705, -1.6185719966888428, 0.4654752314090729, 1.4650155305862427, -0.353738009929657, 0.1029304563999176, -0.12839201092720032, 0.9883658289909363, 0.5602843761444092, -0.18330776691436768, 0.9416272640228271, -0.26616182923316956, 0.5157487988471985, 0.15510962903499603, 0.20672284066677094, -0.08114442229270935, -0.40769466757774353, -0.9592460989952087, -0.04015957936644554, 0.10901878029108047, 0.04230942577123642, 0.005834084469825029, -1.6309850215911865, -0.561631441116333, -0.07395783066749573, -0.4876677989959717, 1.5900994539260864, -0.02304575778543949, -2.2878055572509766, 0.4453238248825073, -1.9236652851104736, -0.8954785466194153, 0.2400108277797699, 1.472288727760315, 0.05976809933781624, 0.26033973693847656, 0.03439962863922119, -0.004904458299279213, -0.4235546886920929, 3.2821731567382812, -0.24173472821712494, 0.6056820154190063, 0.07786917686462402, 1.1722123622894287, 0.004334357567131519, 0.22047144174575806, -0.8133080005645752, 0.07403584569692612, 0.16993077099323273, -0.09831862151622772, -0.2745504379272461, -0.2951886057853699, 1.0217384099960327, 0.3021191954612732, 0.6595038175582886, -0.19343340396881104, 0.730697751045227, 0.42202404141426086, 0.44014647603034973, -0.36273902654647827, 1.4057254791259766, 0.26511746644973755, -1.2572845220565796, 0.9280433654785156, -0.006309389136731625, -0.020094236359000206, -0.052815936505794525, 0.5892437696456909, 1.1003495454788208, -3.96002197265625, 0.8625006079673767, 1.6083576679229736, 0.08250074833631516, 0.08796201646327972, -0.4273529648780823, -0.29951614141464233, 6.060014724731445, -0.007078804075717926, 0.519771933555603, -0.35337117314338684, 0.026951052248477936, -0.43617767095565796, -0.44240620732307434, 0.2517406940460205, -0.08020123094320297, 1.0311118364334106, -0.09693150967359543, 0.2937288284301758, -0.01892942748963833, 0.7498331665992737, 1.0670359134674072, 1.0020670890808105, 0.5595937371253967, 1.582726001739502, -0.1529727727174759, -0.3292151093482971, 0.14765939116477966, 0.012867714278399944, 0.4675999879837036, -2.06998610496521, -0.1448940932750702, -0.033064235001802444, -0.07721192389726639, -0.537814199924469, -0.027970703318715096, 0.20124539732933044, -0.23833225667476654, -0.30280351638793945, -0.21496616303920746, 0.5713273882865906, 1.0038760900497437, 1.4206918478012085, 0.8755730986595154, -0.09116681665182114, 4.265132427215576, 0.06330159306526184, -0.83473801612854, 0.25724124908447266, 0.21598570048809052, -2.556368350982666, 0.23961322009563446, 0.03298461437225342, -0.28364431858062744, -0.5675828456878662, 0.34952396154403687, 0.6501049995422363, -0.1634027659893036, -1.1786940097808838, 2.2006869316101074, 1.3336139917373657, -2.4290525913238525, -0.2849525809288025, -0.6200936436653137, 0.0002452330372761935, -2.342801809310913, -0.835612952709198, -0.07084163278341293, 0.10545112192630768, 0.8178076148033142, 0.4370191693305969, 0.17870819568634033, 0.2998283803462982, 0.05971284583210945, -0.4253867566585541, -0.842909038066864, -2.0334970951080322, -1.3121778964996338, -2.5457165241241455, 0.40157416462898254, 3.3756906986236572, -0.12073466181755066, -1.0871360301971436, -0.17040908336639404, -1.1741911172866821, -0.6737989783287048, -0.09236568957567215, 1.4516136646270752, -0.11569111049175262, -0.17320634424686432, 0.29295849800109863, -0.5863015651702881, 0.08624078333377838, -2.149319887161255, 0.012135802768170834, 0.22052869200706482, -0.03837910294532776, 0.9762369394302368, 0.5947921276092529, 0.006969542242586613, -0.04315227270126343, -0.05992576852440834, -0.2569498121738434, -0.9946044683456421, 0.4223479926586151, -1.7005565166473389, -0.09686321020126343, 0.255108505487442, -0.008139627054333687, 0.18845956027507782, 1.3623090982437134, 0.16629116237163544, -0.2854444682598114, -0.5158839225769043, 0.7901487350463867, 0.8821036219596863, 0.047840096056461334, 0.020057642832398415, 4.291484832763672, -0.2344903200864792, 1.2774003744125366, -0.3352580964565277, -0.06878471374511719, -0.35258224606513977, -0.05005455017089844, -0.03376511484384537, -0.26465779542922974, 0.8013409376144409, -0.2573641240596771, -0.13231322169303894, -0.02018802799284458, 0.19792227447032928, -0.06270284950733185, -0.48568251729011536, -0.6459685564041138, 0.7003214955329895, -0.026444219052791595, -0.08964625000953674, -1.5785335302352905, 0.7899755239486694, -0.18998026847839355, 1.1251826286315918, 2.7214646339416504, -1.1531920433044434, -0.8254311084747314, 0.6369677782058716, 0.3773042857646942, -0.15835434198379517, 2.098099708557129, -0.8624675273895264, 0.8463922142982483, -0.2440808117389679, 0.17557717859745026, -0.10692593455314636, 0.056270066648721695, 0.6836678385734558, -1.1401945352554321, 0.014535000547766685, 1.757417917251587, 0.051298536360263824, -0.5623307228088379, -0.05804960057139397, -0.8513146638870239, 0.19174350798130035, 2.670606851577759, -0.45189204812049866, 0.2504725456237793, 1.693928837776184, 0.18718820810317993, 1.2347705364227295, -1.380893349647522, 0.028750047087669373, 1.3601154088974, -0.205372616648674, 0.49213066697120667, -2.486215591430664, -0.36721140146255493, 0.7245666980743408, 0.9003995060920715, 0.21986939013004303, 0.9644677639007568, -0.09498878568410873, 0.2724476456642151, -0.27757471799850464, -0.647031307220459, -0.5656941533088684, 0.4055940508842468, 0.6787959337234497, 1.5857653617858887, 0.3078345060348511, -0.01750648207962513, -3.099884271621704, 0.03470570221543312, 0.12993402779102325, -0.08198731392621994, 0.9230706095695496, 0.09080156683921814, -0.14413940906524658, 0.25244003534317017, -0.0010256951209157705, 0.029677381739020348, -0.24976703524589539, 0.7941750288009644, -0.1420002430677414, -0.23730814456939697, 1.5063401460647583, 0.882773220539093, 0.0030201643239706755, 0.03906525671482086, -0.7541154026985168, 0.6541889309883118, 0.705147922039032, 1.4512358903884888, 1.482497215270996, 0.15297704935073853, 1.0511776208877563, 1.9954105615615845, -0.23024733364582062, -1.3081443309783936, 0.9828721880912781, -0.022974738851189613, -4.235408306121826, 0.20801524817943573, 0.2791517674922943, 1.005122423171997, 0.10320772975683212, -0.1867409497499466, -0.09179872274398804, 0.5964970588684082, -0.1970781683921814, 0.09882822632789612, -0.03629954531788826, 1.088611125946045, 2.934846878051758, -1.7390817403793335, 0.43121278285980225, -0.0815696269273758, -2.0166165828704834, -0.5055879950523376, 0.29834017157554626, 0.025263890624046326, -2.4019577503204346, -0.8628925681114197, -4.364862442016602, -0.0029420251958072186, -0.24286796152591705, 0.06722001731395721, -1.0107014179229736, 0.5566327571868896, -0.491126149892807, 0.7796655297279358, -1.5919193029403687, 0.013283544220030308, 0.021276332437992096, -0.00263407826423645, 0.030473854392766953, 0.329907089471817, 1.3131346702575684, 2.2258598804473877, 2.314138650894165, -0.532505452632904, 1.2503658533096313, 0.9169355034828186, -0.0905526876449585, -0.42108669877052307, -0.5134214162826538, -0.5456083416938782, 1.0716949701309204, -0.04488784819841385, 0.5283491611480713, -0.6114278435707092, 0.588880181312561, 0.48829036951065063, 0.492691308259964, -0.2285405993461609, -0.2354479879140854, -0.7503323554992676, -0.3818279802799225, -1.1435457468032837, 0.7251201868057251, -0.2693808078765869, -1.3961937427520752, 1.803732991218567, 0.034856442362070084, 0.8880653977394104, -1.2863801717758179, -0.1564207822084427, -0.4329230785369873, 0.06188158318400383, -0.37871092557907104, 1.2700453996658325, -0.3911757469177246, 0.34670206904411316, -0.045261893421411514, -0.13470901548862457, 0.2836616635322571, -0.4087967276573181, 0.01097066979855299, 0.0440337210893631, -0.043102510273456573, -0.42426544427871704, -0.042499035596847534, 1.9497654438018799, 0.14954034984111786, -0.35380107164382935, -0.24830147624015808, 0.06609884649515152, -0.6870531439781189, 0.2430599331855774, 0.19358567893505096, -1.4555237293243408, 0.6210869550704956, 1.3683186769485474, -0.1091524139046669, 0.3807857036590576, 0.10340265184640884, 0.10524800419807434, 0.7470381259918213, 0.042613059282302856, -0.09037353843450546, -0.2095596045255661, -0.21284058690071106, 2.355855703353882, 0.24587836861610413, 1.5879849195480347, -0.3669951856136322, 0.5436751842498779, -0.032810259610414505, -0.8937116265296936, 1.3846924304962158, 0.5658201575279236, 0.04569572955369949, -0.2198866754770279, 0.8332188129425049, -0.7850870490074158, -1.0008292198181152, -0.017097534611821175, -0.0646539106965065, -0.276608943939209, 1.4551092386245728, 1.0873422622680664, 0.5624172687530518, -0.06580200046300888, -0.6027804017066956, 0.40782245993614197, 0.017314627766609192, 0.7225601077079773, 0.006828273180872202, 0.3177792429924011, 1.1174695491790771, 0.2344013899564743, 0.13112781941890717, -1.921704888343811, 0.5512420535087585, -0.4232140779495239, 2.4743857383728027, 0.781706690788269, 0.64940345287323, -0.1276235431432724, -0.15058839321136475, 0.7162768244743347, -0.22906945645809174, 1.9664556980133057, -0.0448785163462162, 1.185067057609558, -1.0092672109603882, 0.844467282295227, 0.08970672637224197, 1.5572147369384766, 0.14723479747772217, -1.0999056100845337, -0.7793235182762146, 0.24421454966068268, -0.06395073235034943, -0.7480176687240601, -0.6278814673423767, -0.0681554526090622, -0.48046571016311646, -0.8333029747009277, 0.13902296125888824, 0.3002829849720001, -1.5550791025161743, 0.8308951258659363, 0.4930700957775116, 0.062407124787569046, 0.1145152747631073, -0.13467757403850555, -0.047001197934150696, 0.14158181846141815, 0.12175650149583817, -0.0579889640212059, -0.9262872934341431, -1.8610650300979614, 0.1834861934185028, 0.03542746603488922, -0.040891651064157486, 0.58236163854599, -1.482275128364563, -0.14007677137851715, 0.03215863183140755, 0.19385094940662384, 1.9605671167373657, -0.5531213879585266, -0.04560894891619682, 0.17624473571777344, 0.07804927974939346, -0.0908026173710823, -0.9305290579795837, -0.9204409122467041, -0.3458043336868286, -0.5171460509300232, 0.19721446931362152, 0.47269904613494873, 0.16442729532718658, -1.3050100803375244, -1.8236362934112549, 0.008336285129189491, -0.8778125047683716, 0.5046382546424866, 0.9916731119155884, 0.593207061290741, -0.04051494225859642, 0.24280235171318054, 0.7992647886276245, -2.2296602725982666, -0.1556524783372879, -0.8503082394599915, -0.11854985356330872, -0.5535645484924316, -0.9994786977767944, -0.7202552556991577, 0.8698686361312866, -0.9949544072151184, 0.10016124695539474, 0.08882118761539459, 0.8583472371101379, -0.0005518959369510412, 0.860970139503479, -0.49673572182655334, -0.5857544541358948, 0.5294370055198669, 0.09730961918830872, 0.41050440073013306, 0.17713259160518646, -0.1708470731973648, -0.5204184651374817, 1.6572374105453491, -0.568669319152832, 0.16514559090137482, -0.24080494046211243, -0.00017582550935912877, 0.23844417929649353, -0.0279901921749115, 0.03377816453576088, -0.4145885407924652, 0.42469239234924316, -2.729980230331421, 2.0221657752990723, 0.09444747865200043, -0.5531710982322693, 0.03152655065059662, 0.7961639165878296, -0.1005236878991127, 1.1841977834701538, 1.0450559854507446, 0.25950902700424194, 1.65407133102417, 0.5448261499404907, 0.7429105639457703, 0.0010173374321311712, 1.098086953163147, 0.2938084304332733, 0.8080891370773315, 0.5029069185256958, 0.10236162692308426, 2.9149715900421143, -0.836000382900238, -0.7567518353462219, -0.04422072321176529, 0.45084649324417114, -0.01167692057788372, -0.0011878631776198745, -0.6863207221031189, -0.44385525584220886, 0.5870376229286194, 0.5830861330032349, -0.8575820922851562, 0.7331447005271912, -0.6482118368148804, 2.4027624130249023, 0.9764196872711182, -0.18936996161937714, 0.7530697584152222, 1.6529896259307861, 0.16582739353179932, 0.004192407708615065, 0.11686883121728897, -0.09926917403936386, -1.4776290655136108, -1.074103593826294, -0.12321784347295761, -0.04829884320497513, -0.660578727722168, -0.17916204035282135, 2.756899356842041, -0.35748887062072754, 0.36677342653274536, -0.8725548982620239, 1.222765326499939, -0.5469682812690735, -0.20022210478782654, -0.07971978187561035, 0.18201346695423126, 0.05946498364210129, 0.854012668132782, -0.03782554715871811, 0.07222847640514374, 0.12126913666725159, -2.0005228519439697, -0.26998162269592285, -0.3453853130340576, -0.5732050538063049, -0.19050489366054535, -1.4274287223815918, -2.292248487472534, -1.228134274482727, -0.738007664680481, 1.159761905670166, 2.005796432495117, 0.4920658767223358, 0.038599174469709396, -0.7769426107406616, 0.7123469114303589, -1.8825721740722656, -0.0741986334323883, 0.7525733709335327, -0.5661430358886719, 2.0247726440429688, 0.529891848564148, -0.005066463723778725, -0.5864160060882568, 1.119092345237732, 0.38984087109565735, -0.030503971502184868, 1.247225046157837, 0.12469420582056046, -0.5275996327400208, -0.07702896744012833, -3.562725782394409, -1.900631308555603, 0.7056753039360046, 0.001324823359027505, 0.07161755859851837, 0.01981920748949051, -1.8128280639648438, -0.23120152950286865, 1.3614656925201416, -0.3561230003833771, 0.9383436441421509, 0.4411253333091736, -0.6283693313598633, 0.2992541491985321, 0.5171154141426086, -0.06603806465864182, -2.350428342819214, 0.2541808784008026, 0.17417702078819275, 1.215153694152832, -0.7657862305641174, 0.4590607285499573, -0.01963741146028042, 0.05106115713715553, -0.024434395134449005, 0.9215524196624756, 0.48882755637168884, -0.6584857106208801, -0.5813736319541931, 0.8051072359085083, -0.5350973606109619, -0.20991963148117065, 0.3396441340446472, 0.650567889213562, -0.20662468671798706, -0.46314504742622375, -0.7823758721351624, -0.18978776037693024, -0.04833139479160309, 0.19013947248458862, 0.5041614174842834, 1.804041862487793, 0.7711649537086487, 0.2807307541370392, -1.937997579574585, 0.5744585990905762, -0.05934389680624008, -1.0533722639083862, -0.23784057796001434, 0.23653548955917358, 2.3445241451263428, -0.04282861202955246, -0.02787913754582405, 0.15777167677879333, -2.9190635681152344, -0.02110917866230011, 0.1536388248205185, 0.09363092482089996, -0.9543060064315796, 1.5097947120666504, 2.0704777240753174, -0.2854713499546051, -0.041836366057395935, 1.5544703006744385, -0.11587212234735489, -0.5170774459838867, -0.2711736261844635, 1.2196476459503174, 0.4510703682899475, -0.18378561735153198, -0.37486302852630615, 2.348048448562622, 0.8509658575057983, -0.043962929397821426, -0.09598188102245331, 0.7785370349884033, -0.5112394690513611, 0.46182164549827576, -0.3317263424396515, -0.14691521227359772, 0.026005754247307777, 0.21994833648204803, -1.2871707677841187, -0.056390129029750824, -0.08984784781932831, -0.12176720052957535, -0.2231384813785553, 0.23420454561710358, -0.6153751611709595, -0.3389740586280823, 0.49781590700149536, 1.7854928970336914, -0.8286357522010803, -0.47184982895851135, 0.8983160257339478, 0.24659617245197296, 0.4106028974056244, -0.11907003819942474, -0.3926962912082672, -0.030492747202515602, 0.05746123194694519, -0.08016564697027206, 0.40892231464385986, 0.019398659467697144, -0.42628562450408936, 0.9383017420768738, 0.2819812595844269, 0.044153522700071335, -0.6505839228630066, 0.060606423765420914, 1.249485969543457, 0.25468704104423523, 2.058279275894165, 1.6658129692077637, -0.15546654164791107, 0.19813935458660126, -1.299633264541626, -0.0084525216370821, -0.9009218215942383, -0.13782081007957458, 0.04035668447613716, 0.08969362825155258, 0.6482704877853394, -0.37151458859443665, 1.8999332189559937, 2.375333309173584, -0.24378226697444916, 0.08241178840398788, -0.6667454838752747, 0.2676214575767517, -0.39601069688796997, -0.777320384979248, 1.530140995979309, 0.22791074216365814, 0.011837896890938282, 0.6143869757652283, -1.231400489807129, -0.4032787084579468, 0.7603772878646851, 0.00456439470872283, 1.2905224561691284, -0.9847093224525452, -0.17218941450119019, -0.16695664823055267, 0.33274075388908386, -0.5664953589439392, -0.4732072651386261, 0.1255045384168625, 0.6810636520385742, -0.38576531410217285, 0.11289530247449875, -0.27907267212867737, 0.3065907955169678, 1.0367233753204346, -0.04717041924595833, -2.141993522644043, 0.0034511403646320105, -2.376354694366455, 0.987347424030304, 0.40114298462867737, -0.45132070779800415, 0.6550247669219971, 0.1837024837732315, -0.6819998621940613, -0.02151309698820114, 0.05863679572939873, 0.37473395466804504, -0.05229007080197334, 0.10792475938796997, -0.03380344808101654, 1.2024301290512085, 0.012003455311059952, 0.26128950715065, -0.12212681025266647, -0.23315666615962982, -0.5949016809463501, 1.1790082454681396, 1.5281273126602173, -0.15655559301376343, 0.8570511937141418, -0.7311723232269287, -5.296860218048096, 1.1942805051803589, -0.5529170632362366, -2.1145877838134766, -0.010536772198975086, -0.01414018590003252, -0.15713635087013245, -0.27064213156700134, 0.3177793622016907, -0.07465779781341553, 0.2790036201477051, 0.32609841227531433, -0.0008984585874713957, 1.677331566810608, -0.06518042087554932, 0.35317468643188477, -0.286909818649292, 1.7604128122329712, -2.189209461212158, -0.1401485800743103, -0.15312837064266205, 0.13603496551513672, -0.29791611433029175, -1.0649839639663696, -0.1902008354663849, -0.4602483808994293, -0.39050915837287903, 0.09339132905006409, -0.042538825422525406, 0.33939051628112793, -0.21044664084911346, -2.470418691635132, -1.8613795042037964, -1.6542589664459229, 0.012798398733139038, -0.10789517313241959, 0.2841469645500183, -3.3856122493743896, -0.24530327320098877, 1.559130072593689, 0.46935024857521057, 0.9265232682228088, 0.19239391386508942, -0.5509180426597595, -0.7550517916679382, 0.22766776382923126, 0.2344379723072052, 0.45293793082237244, -0.1992737352848053, -0.5712252855300903, -0.09915254265069962, -0.5498896837234497, -0.16138432919979095, 0.18843096494674683, -0.51548171043396, 0.42390933632850647, -0.020878823474049568, -3.1864869594573975, -0.4058992564678192, 0.013216204009950161, 0.07837861031293869, -0.12998828291893005, -1.8502826690673828, -0.6715171337127686, 1.0348883867263794, 0.06674827635288239, 0.7849804162979126, -0.29195210337638855, -1.5472095012664795, 0.025131355971097946, 1.996691346168518, -0.25689929723739624, 0.31494319438934326, 0.5007192492485046, -2.4421842098236084, 0.2073879987001419, -0.7882052659988403, -0.19062595069408417, -0.23092696070671082, -0.4159601032733917, 2.624054431915283, -1.6583118438720703, 0.27341535687446594, 1.5337740182876587, -0.045325491577386856, -0.33687636256217957, -3.895256280899048, 4.144798755645752, 2.2243294715881348, -0.28769803047180176, -0.008272238075733185, -1.06169593334198, -1.2658668756484985, 0.4654183089733124, -0.3101668953895569, -0.7853639721870422, 0.5904013514518738, -0.014464709907770157, -0.5121661424636841, -0.02632301114499569, -0.6640071868896484, 0.08728977292776108, 0.590143620967865, 0.5579988956451416, -0.31130656599998474, -0.6789892911911011, -0.10372631251811981, -1.9072105884552002, -0.17794948816299438, -0.3324245512485504, -0.03990837186574936, -1.5059140920639038, -0.062149014323949814, -0.22590303421020508, 0.045669395476579666, -0.007852240465581417, 0.8076003789901733, 0.22686456143856049, -0.038220301270484924, 0.34269267320632935, 1.1152329444885254, 0.04002160206437111, -2.401401996612549, 1.135418176651001, -0.01624695584177971, -0.7132521867752075, -0.22277170419692993, 0.32389238476753235, -0.8647312521934509, -0.3899219334125519, -1.0475424528121948, 0.3184923231601715, -0.06996697187423706, 1.1044632196426392, 0.02199479565024376, 0.024154257029294968, 1.142498254776001, -1.9137619733810425, -0.9948667287826538, 0.3753852844238281, 0.18793362379074097, 0.11503621935844421, -1.7932062149047852, 1.0158159732818604, 1.3006410598754883, 0.13390059769153595, 0.30771446228027344, 0.18783243000507355, 0.527681291103363, -2.4087629318237305, -0.8151808977127075, 0.5498588681221008, 0.3738418221473694, 1.0518126487731934, -0.05476287752389908, -0.29740336537361145, -0.21905438601970673, 0.009325094521045685, -0.3811175525188446, -0.0013912372523918748, 1.7787500619888306, 0.5449770092964172, -0.35745251178741455, 0.4813953936100006, -0.7152631878852844, -0.021285900846123695, -1.0148452520370483, -0.039924584329128265, 2.307262659072876, -0.14588403701782227, -0.5312134623527527, 0.2855043113231659, 0.1481156051158905, 0.5613712072372437, -0.6925026774406433, 0.054077230393886566, 0.7283429503440857, -0.07426149398088455, -4.740728378295898, 2.285773754119873, -3.318920373916626, 0.24298414587974548, 1.2313518524169922, -0.18114741146564484, 0.05191817134618759, -2.0759270191192627, 2.2781364917755127, -0.011298110708594322, -1.229559302330017, 0.048772215843200684, 0.004529494792222977, 0.0001259935525013134, 0.22820912301540375, -0.21311938762664795, -0.1046585664153099, -0.5970494151115417, -0.7147109508514404, 0.03768046200275421, -0.028425853699445724, 0.10996938496828079, -1.3780078887939453, -0.3238185942173004, 3.4720866680145264, -0.23059938848018646, 0.02123352512717247, 0.5863997936248779, 0.013997131958603859, -0.026641706004738808, 0.24968969821929932, -0.4765191972255707, 0.13218504190444946, -0.03082098811864853, 2.354419469833374, -0.3083224594593048, 0.22666089236736298, 0.4646962285041809, -0.9086130261421204, 0.8678054809570312, 0.1925860345363617, 0.48466697335243225, 0.014134490862488747, -0.30253955721855164, 1.0986374616622925, 0.014051379635930061, -0.4704812169075012, -0.404389351606369, -0.17067843675613403, -0.43251362442970276, 0.302584171295166, -0.8634653687477112, 0.12105827033519745, -0.2946998178958893, -0.2405039519071579, -2.813023567199707, -0.24854792654514313, -1.3960528373718262, 0.1469552367925644, 0.7285488247871399, 0.6022917032241821, -0.48147818446159363, 1.3076599836349487, -0.04589090496301651, -0.2545757591724396, -0.4947177469730377, 0.36529484391212463, -0.7259156703948975, -0.1833212524652481, 0.736801266670227, 1.3640532493591309, -0.6545527577400208, -0.38190770149230957, -0.13630279898643494, 0.4628594219684601, -2.820230007171631, 0.17894090712070465, 0.033375825732946396, 0.04392601177096367, -0.300361305475235, 1.8920284509658813, -0.26450279355049133, 0.494231641292572, 0.5052772164344788, -0.09714914113283157, 0.4062095880508423, 0.10621555149555206, 1.6555399894714355, -1.3415290117263794, 1.242523431777954, 0.017546415328979492, -0.401027649641037, -0.30250024795532227, 0.14173319935798645, 0.7207244038581848, -0.023821726441383362, -0.9985682368278503, -0.18889929354190826, 0.6672585606575012, 0.16574597358703613, 0.3225339949131012, 0.9174908399581909, 0.09303637593984604, 0.9040456414222717, -1.4906891584396362, 0.2976880669593811, 0.4331991374492645, 0.9258987903594971, -0.3875260651111603, -0.0267082080245018, -0.18092453479766846, -1.1461797952651978, -0.24065205454826355, -0.7639105319976807, 0.7028431296348572, 0.3641242980957031, -0.1987907737493515, 1.2868574857711792, -0.005138499196618795, -0.044475309550762177, 0.019947586581110954, -0.07441425323486328, 0.8353863954544067, 1.1782582998275757, 0.3139730393886566, -0.5671860575675964, -0.38383764028549194, -7.113269805908203, -0.24752479791641235, -0.22037287056446075, 1.9439421892166138, 0.11041227728128433, 0.09520705789327621, -0.0001710981159703806, 0.012791359797120094, 0.48718884587287903, 0.019732680171728134, -0.45630091428756714, -0.09742741286754608, -0.0008513788925483823, 0.28923195600509644, 0.19772113859653473, 0.08422578126192093, -2.699965715408325, -0.5306045413017273, 0.12431418895721436, 0.35132259130477905, -1.1756700277328491, -0.7164582014083862, 1.4193854331970215, 0.23266594111919403, -0.3014513850212097, -3.3720486164093018, 1.4372732639312744, 0.057532425969839096, 0.07122983783483505, -0.08242268860340118, -0.2340128868818283, -0.4063672721385956, 1.1571344137191772, -0.9266117811203003, 0.15932676196098328, -0.4540945589542389, -0.03895958140492439, 0.13405922055244446, -0.8166202306747437, -2.1716299057006836, -0.24567759037017822, 0.11085381358861923, -0.2299143522977829, -2.1237168312072754, -0.7768634557723999, 1.0496670007705688, -0.04371275007724762, -0.8059415221214294, 0.1898394525051117, 0.3730393052101135, 0.1661667823791504, -0.6619900465011597, -0.11699683964252472, 0.015041378326714039, -0.07502177357673645, -1.1783877611160278, -0.18983815610408783, -0.2229912430047989, -0.04153410717844963, -0.7099289894104004, 0.5313403010368347, -0.4837983250617981, -3.154447078704834, 0.07596047222614288, -0.7603036165237427, 0.24402070045471191, -0.21023859083652496, -0.15961334109306335, 0.6387276649475098, -0.03076961450278759, -0.3640134632587433, -0.0023842882364988327, -0.13332952558994293, 0.21718883514404297, -0.296189546585083, 1.0387747287750244, 0.9846627116203308, -1.0002704858779907, -2.7394485473632812, 0.3401453495025635, 1.0103250741958618, 0.4465397596359253, -0.23755094408988953, -0.16445763409137726, 0.15830667316913605, 0.3432905673980713, -0.4398268461227417, -0.5667915940284729, 0.21081987023353577, 0.2720677852630615, -0.06105455383658409, 0.05441553518176079, 0.057713333517313004, -9.247939487977419e-06, -0.5489564538002014, 0.011029228568077087, -1.9382119178771973, -1.551379680633545, 0.9937514662742615, -0.21759699285030365, -1.519118309020996, 0.2550603151321411, 0.08737211674451828, -0.0640515685081482, 0.3274490535259247, -0.17393293976783752, 0.03153727948665619, 0.88359534740448, -0.1875986009836197, 0.8169891238212585, 0.04021010175347328, -0.3527594208717346, 0.015169696882367134, -0.44808295369148254, -0.01681630313396454, 0.10276000946760178, 0.9014561176300049, -1.7133324146270752, 0.7042585015296936, 0.15403975546360016, -0.17529140412807465, 0.881118893623352, 0.07911867648363113, 1.622762680053711, 0.8930784463882446, -0.3856217861175537, 0.722046434879303, 1.4194470643997192, -0.6177109479904175, 0.10855916142463684, -0.21717704832553864, -0.6599475741386414, -1.2072840929031372, 3.731761932373047, 1.2784188985824585, -1.897300362586975, 0.1512828767299652, 0.035115331411361694, -2.773573398590088, -0.2291652411222458, -0.09199743717908859, -0.4278733730316162, 0.08701401948928833, -0.3313702344894409, -1.7826282978057861, 0.07636227458715439, 0.5129328370094299, 0.23696143925189972, 0.07398410141468048, -0.05732899159193039, 1.3280590772628784, 0.17340192198753357, -0.12722405791282654, -0.2403242290019989, 0.5926852226257324, -0.2044714093208313, -1.6213241815567017, 0.09255535155534744, -0.4403146207332611, 1.4653551578521729, 0.2777736186981201, 0.0034866577479988337, -0.840617835521698, -0.10767612606287003, -0.3355255722999573, -0.044917818158864975, -0.04981105402112007, -1.2715420722961426, -0.019701039418578148, -0.7456305027008057, 0.09576056897640228, -0.500001072883606, 0.08802194148302078, -0.5633360743522644, -1.045592188835144, 0.5654996037483215, 0.7611666321754456, 0.40097576379776, -0.18109062314033508, -4.838802337646484, 3.5311288229422644e-05, -1.153706669807434, -0.005567660089582205, -0.32363948225975037, 0.7003375291824341, -0.4453276991844177, -0.11610973626375198, -0.05208008736371994, 0.11088641732931137, -0.1805018186569214, -2.5111732482910156, -0.6258541345596313, -1.5806657075881958, 0.00696475338190794, 1.8514853715896606, -0.8816277384757996, 0.6004474759101868, -0.19843140244483948, -0.0034163491800427437, 0.18881256878376007, -2.117755174636841, -1.1525843143463135, -0.35383448004722595, 0.2988567054271698, -1.1975178718566895, 0.6660809516906738, -0.022966885939240456, -0.49548080563545227, 0.08947180956602097, 0.4207855761051178, 0.32690373063087463, 0.21952959895133972, 0.8085908889770508, 0.365224689245224, 0.12797051668167114, -0.0030124669428914785, 0.05519646406173706, -0.20214423537254333, 0.09193571656942368, 0.10641052573919296, 0.014696110971271992, -0.7149688601493835, 0.2866959571838379, 1.248143196105957, -0.2669886350631714, -0.2171323299407959, 0.2839919626712799, -0.06613238155841827, 0.0038534696213901043, 0.3647887408733368, -0.6605379581451416, -0.31445980072021484, 0.06645532697439194, 0.5484102368354797, -0.013766816817224026, -0.7025547027587891, 0.011986283585429192, -0.10064937174320221, 0.23230138421058655, -0.02386370673775673, 1.7742769718170166, 4.43358039855957, -0.5558895468711853, -0.12080930173397064, 0.035301804542541504, 0.08519585430622101, -0.28291672468185425, -3.655695676803589, -0.34929996728897095, 0.39709845185279846, -0.7221477031707764, 0.5105177164077759, 0.026448382064700127, 0.24239283800125122, 0.302150160074234, 0.05066012963652611, -0.00653851218521595, 0.64536452293396, 0.18963557481765747, -0.8967887759208679, 0.001196616212837398, 0.9649957418441772, 0.4480472207069397, 0.288976788520813, 0.042711541056632996, 0.9163023829460144, -0.03991974517703056, 0.8660098910331726, 0.16900642216205597, 0.8754718899726868, -0.2169203907251358, -0.029889985918998718, -0.07768841087818146, 0.2708534896373749, -0.47284677624702454, 1.7652983665466309, 0.24658896028995514, -0.2338487058877945, 0.8007564544677734, 0.026294158771634102, 0.41858941316604614, 0.5529570579528809, -0.1403665691614151, -0.7500995397567749, -0.015021556057035923, -0.09699508547782898, -5.337703704833984, 0.21316546201705933, -0.7581816911697388, 2.2351160049438477, -0.8542683124542236, -0.37311384081840515, 2.531320333480835, 0.10624263435602188, -0.7131739258766174, -0.6241429448127747, 0.0013030298287048936, 1.8669475317001343, -0.1381651759147644, 0.018693527206778526, 0.09065894037485123, 2.5780625343322754, -0.638104259967804, 0.6310208439826965, -0.31244170665740967, 0.48216864466667175, 1.399893879890442, -1.0348918437957764, -0.08096661418676376, 1.7679634094238281, 0.2696090042591095, 0.5450304746627808, -1.0289983749389648, 1.795830249786377, -0.2735113203525543, -0.47970935702323914, -0.8694196343421936, -0.6262410283088684, -2.7264232635498047, -1.157368540763855, -0.2992897629737854, -0.4694387912750244, -0.01080155000090599, -2.270838975906372, 2.1598703861236572, -0.04376918822526932, 0.7762935757637024, 0.09548675268888474, -0.5234878659248352, -0.3652561604976654, 0.06419414281845093, -0.3707478940486908, 0.2549225091934204, -0.2566053569316864, 0.05255590006709099, 1.1153862476348877, -0.01994229666888714, -0.13027146458625793, -2.1042399406433105, -0.30256277322769165, 0.4925498068332672, 0.30464330315589905, 0.000994406291283667, 1.2540066242218018, -0.38551703095436096, 0.010111006908118725, 0.32014402747154236, -0.33205997943878174, -0.0058746179565787315, -0.08092391490936279, -0.0931595116853714, -0.1812414675951004, 0.1349601000547409, 0.043968938291072845, -0.10488416999578476, 0.1524391919374466, 0.339857816696167, -0.5150747895240784, -1.2107471227645874, -0.20172153413295746, -0.1482626348733902, -0.33081191778182983, 0.19615823030471802, -0.7812711596488953, 0.4522274136543274, 0.07446708530187607, -0.4711413085460663, -0.8270335793495178, -1.4341206550598145, 1.2039819955825806, -0.8550064563751221, -0.11117544025182724, -1.5588781833648682, -0.01075536385178566, 0.06497343629598618, 0.7469678521156311, -1.4170057773590088, 0.22756712138652802, -2.0580124855041504, -1.359333872795105, -0.013616016134619713, 0.134622260928154, 0.42847350239753723, 1.3698819875717163, -0.5699397921562195, 1.0602500438690186, 1.3453296422958374, 0.4840617775917053, -0.0023960487451404333, -1.1443227529525757, 0.23737733066082, -0.2533992826938629, 0.3273935914039612, -0.08509325981140137, -0.27659255266189575, 0.393795371055603, -1.3985813856124878, 0.3755890429019928, -1.6039384603500366, 2.6273181438446045, 0.014531529508531094, 1.2037991285324097, -0.06164426729083061, -1.3099268674850464, 0.926180899143219, -1.7857310771942139, 0.6173785924911499, 0.0158633254468441, -0.6234745979309082, 0.24396172165870667, -1.5742186307907104, -1.767177939414978, 0.852506697177887, -0.7062692642211914, -0.17568722367286682, -0.8471579551696777, 0.48707112669944763, 1.025389313697815, 0.2788618803024292, 0.6114609241485596, 0.19064080715179443, -0.07044777274131775, 0.1847839504480362, -0.9465370178222656, -0.0024428574834018946, -0.07660290598869324, 0.27394726872444153, 0.3488100469112396, -1.2491025924682617, -0.003648288082331419, 1.9029159545898438, -0.06818117946386337, -0.22935578227043152, 0.630570113658905, -1.5517618656158447, -1.1419061422348022, 0.02283564768731594, 1.1260032653808594, 0.02831992134451866, 0.09321750700473785, 0.04973556846380234, -0.07615441083908081, 0.19174018502235413, -0.862525224685669, 1.0004552602767944, 0.42526009678840637, -0.021865401417016983, -0.4537803828716278, 0.29558485746383667, -1.4388943910598755, -0.016044357791543007, -0.913281261920929, 0.25705593824386597, -2.703953266143799, -0.48150402307510376, -0.29439404606819153, -0.7162023186683655, -0.11193181574344635, 0.007614449132233858, -0.8289395570755005, 0.009697329252958298, 0.17481784522533417, -0.42942067980766296, -2.5274322032928467, -0.6945486068725586, 1.310115933418274, 0.7528474926948547, -0.6531211137771606, -2.6203010082244873, -0.03447316586971283, -0.07631202042102814, 0.10222173482179642, -0.374543160200119, -0.7775174975395203, 0.13003063201904297, -0.9580807685852051, 0.5229482054710388, 0.49785560369491577, 0.3824499547481537, -0.5494207143783569, -0.922191858291626, 0.07298643887042999, 0.08003943413496017, 0.6023558974266052, 1.1451280117034912, -0.05566432699561119, 0.4924956262111664, -1.1527997255325317, -0.36142826080322266, -0.4798760414123535, 0.4193045198917389, -0.042595718055963516, -0.7308411002159119, 0.0553121380507946, -0.2124456763267517, 1.5039650201797485, -0.7010939121246338, -0.11518597602844238, 2.4746131896972656, -0.4467991292476654, 0.1407776176929474, 0.03204433619976044, 0.5440770387649536, -0.22195541858673096, 0.9000965356826782, -0.1775464117527008, -0.9983932971954346, 0.017652448266744614, 0.3339287340641022, 0.13276639580726624, -0.1299980878829956, -0.8045963048934937, 0.10422322154045105, 0.8363036513328552, -1.0187526941299438, -0.7051156163215637, -0.010113555006682873, 0.5889442563056946, 3.3496716022491455, 3.339416980743408, 0.045514147728681564, -0.4444529116153717, 0.5539411902427673, 0.26445773243904114, -0.018980149179697037, 0.07587240636348724, -0.921097457408905, 0.7744355797767639, 0.27403151988983154, -1.3208616971969604, -1.2495924234390259, 1.1296530961990356, 1.008040428161621, -0.056777115911245346, -1.193649172782898, -0.2802063524723053, 0.44051986932754517, 0.11464884132146835, -0.4292975962162018, 0.4991797208786011, 0.6027068495750427, -0.10840858519077301, -1.051409125328064, 0.48565196990966797, 0.2522348463535309, -0.8751401305198669, -0.1107008084654808, -0.4580676257610321, 0.8988405466079712, 0.07128962129354477, 2.290936231613159, 0.05103016272187233, -0.3922889232635498, 4.994496822357178, -0.2905433475971222, 0.08368135243654251, 0.018901092931628227, 0.10090965032577515, 1.2992655038833618, 0.7479521632194519, -0.3592393696308136, -0.13163895905017853, -3.5296900272369385, 0.6972331404685974, 4.681064128875732, -0.1925886869430542, -1.0227134227752686, -0.6137992739677429, 0.2285940945148468, 1.1283667087554932, -0.04265376925468445, 0.7108281850814819, -0.13941022753715515, 0.3538695275783539, -0.9435621500015259, 0.1301184594631195, 1.285649061203003, -0.5694184303283691, 1.2386302947998047, 1.9196523427963257, 0.07571156322956085, -0.0035722116008400917, -0.3415949046611786, 2.564810276031494, -0.24566033482551575, -0.10704021900892258, 0.6844779253005981, 3.2064857482910156, 0.17375853657722473, 0.0856383815407753, 0.26503288745880127, -0.010159976780414581, -0.2982620596885681, 0.04415834695100784, -1.3403100967407227, -0.5287593603134155, -0.01677497662603855, -0.0010691964998841286, 0.7029078006744385, 0.9551602005958557, -1.5111140012741089, 1.4762930870056152, -0.7907313704490662, -0.22921964526176453, 0.10286039859056473, -0.7247855067253113, 0.10069569945335388, -3.9989395141601562, -0.010409404523670673, -0.01843256875872612, 0.1900438666343689, -0.1345600038766861, 0.6698137521743774, 0.09235383570194244, 0.014095287770032883, 0.2170111984014511, -0.9144052267074585, 0.5150130987167358, 0.6035356521606445, -0.16893360018730164, 0.05219033360481262, -0.8065323233604431, 0.12472599744796753, -0.11530445516109467, -0.15739186108112335, -0.5670188069343567, -2.7228870391845703, 0.3897566795349121, -0.3868195712566376, -0.006055685691535473, -0.5277389883995056, 0.39154312014579773, 0.06685841828584671, -0.024957306683063507, 0.568192720413208, 0.06862566620111465, 0.7503783106803894, 0.3782355785369873, -0.6036685705184937, -1.8304250240325928, -0.833989679813385, -0.5618341565132141, 2.589304208755493, -1.9967151880264282, -1.5880745649337769, 1.4548412561416626, 0.03103700652718544, -0.07902135699987411, -1.2218760251998901, 0.004347427748143673, -1.4858152866363525, 0.6222808361053467, 1.5540581941604614, -0.0554756335914135, -0.629716694355011, 1.9090574979782104, 0.4953126609325409, 0.14880532026290894, -0.10360510647296906, -1.5574592351913452, -1.9547187089920044, 0.05945444852113724, -0.033187560737133026, 0.02676657773554325, 0.1589922308921814, -0.06371476501226425, 0.03192409127950668, 0.041053228080272675, -2.213430643081665, 0.6380311250686646, -0.7921087741851807, -1.4785195589065552, -0.12461256235837936, -0.38659244775772095, 0.10657451301813126, -0.31116652488708496, 0.3602052330970764, 0.32495182752609253, -0.011060237884521484, 0.8560425043106079, -0.0759689062833786, -0.08881405740976334, 1.3929818868637085, -0.09784257411956787, -1.1013884544372559, 0.9330321550369263, -0.30973726511001587, -0.027283187955617905, -0.5381064414978027, -0.04614504799246788, -0.2770569622516632, 0.21577034890651703, -0.056706689298152924, 0.5016627311706543, -0.41671228408813477, 0.9378225207328796, 0.35228604078292847, 0.004611701238900423, 1.0401204824447632, -0.09190300107002258, 0.7958001494407654, 0.8387284278869629, -0.05658561736345291, -0.4887787997722626, 1.0768159627914429, 0.0973735973238945, 0.055328480899333954, 1.5082491636276245, -2.750516653060913, 1.1774592399597168, 0.8673736453056335, 1.5974676609039307, 0.3571230173110962, -0.007184035610407591, 0.38048529624938965, -0.007342809811234474, -0.07063046842813492, -2.106523036956787, 0.016894526779651642, 0.17792527377605438, -0.0014103003777563572, 0.11592657119035721, -0.12736313045024872, -0.27736014127731323, 1.204916000366211, 0.3121621310710907, 0.6510663032531738, 0.9221259951591492, -0.8646273612976074, -0.6850736141204834, 0.12973499298095703, 0.0837201327085495, 0.36513590812683105, -0.17358848452568054, -0.004660068545490503, 0.9430479407310486, -0.034371230751276016, 0.004499278962612152, -0.9309473633766174, 0.1362874060869217, -0.21091650426387787, 0.21986143290996552, -0.4857676327228546, -0.7014132142066956, 0.4245258867740631, 0.561788022518158, 0.04479930177330971, -1.5553019046783447, -0.9371563196182251, -0.3323342800140381, 0.05983026698231697, 0.18526148796081543, 0.33451181650161743, 0.3199656307697296, 5.926647186279297, 0.479477196931839, 4.079495429992676, -0.3256587088108063, -0.8733027577400208, -0.29852625727653503, 0.5831921100616455, 0.8915870189666748, 0.8154321312904358, -0.7590146660804749, -0.5219365358352661, -3.3757874965667725, 0.6303086876869202, -0.39806511998176575, 1.1813932657241821, -0.7627155780792236, -1.058135986328125, 0.5233268737792969, 0.10112942010164261, -0.12103481590747833, -0.515976071357727, -0.010557027533650398, 1.5262360572814941, -1.3381329774856567, -0.34386172890663147, 0.024053635075688362, 1.302762746810913, 1.4114807844161987, 0.1286340057849884, -0.2053578943014145, -0.47829002141952515, 0.7466812133789062, -1.4111417531967163, -0.21199701726436615, -0.15840202569961548, -0.20625944435596466, -0.005070019047707319, -1.0138351917266846, 0.44843077659606934, -0.1511395424604416, -0.05606315657496452, 0.10202078521251678, 0.10332365334033966, 0.024106968194246292, -0.48552465438842773, -0.03871255740523338, 1.2344728708267212, -0.8059866428375244, 0.044929925352334976, -0.015777254477143288, -0.3327418267726898, -0.30650845170021057, 0.19363196194171906, -1.1513878107070923, 0.22147470712661743, -0.12614934146404266, -0.8677205443382263, 0.014827786013484001, -0.8899040222167969, 0.005864643957465887, -0.1648787409067154, -0.6897845268249512, 1.1450655460357666, 0.2606147229671478, -2.3920352458953857, -1.0592820644378662, 1.1206547021865845, -0.43049025535583496, -0.33239448070526123, 0.5927368998527527, -0.4087446331977844, 1.2105704545974731, -0.7088173627853394, -0.4206962287425995, 2.2432785034179688, 0.0031386835034936666, 0.02199571393430233, -0.5171679854393005, -0.38360875844955444, 0.6455889940261841, -0.9679955244064331, 1.4608893394470215, 2.1569886207580566, -1.0558596849441528, 0.1373470276594162, -0.04203812777996063, -0.22112305462360382, -0.6819552183151245, -0.646629810333252, -0.2756672203540802, 0.00296738906763494, 0.2061668038368225, 0.1312960833311081, -0.027289794757962227, -0.5955817103385925, -0.1764078140258789, -1.656232476234436, 0.4199245274066925, 2.949965476989746, 1.7072553634643555, 0.28379568457603455, 0.1339634507894516, -0.3013043999671936, -0.612144410610199, 1.8442665338516235, 2.4485137462615967, -1.4313552379608154, 0.6249562501907349, 0.631584644317627, 0.22664526104927063, 0.31903088092803955, 0.48002034425735474, 0.23506252467632294, -0.9644860625267029, 0.16280360519886017, 0.1662141978740692, 0.027781963348388672, -0.007611927576363087, 0.23387552797794342, -0.28637486696243286, -0.49705028533935547, 0.21771450340747833, 0.048177849501371384, 0.06056727096438408, 0.5335888862609863, -1.5422511100769043, -1.2209396362304688, 0.08046615868806839, 0.4781278669834137, -0.40640339255332947, 0.20557163655757904, -1.1169692277908325, -0.8161720037460327, 0.371247261762619, -0.029047831892967224, -0.9771010875701904, 2.0629773139953613, -0.14392484724521637, 0.42063528299331665, -0.01612427458167076, 0.4579277038574219, -0.37457016110420227, -0.2026912271976471, 1.6887900829315186, -0.3150736391544342, 0.4643875062465668, -0.44110438227653503, 0.47955915331840515, 0.32060670852661133, 0.25086331367492676, 0.7851729989051819, 0.2757395803928375, -0.3439159095287323, -0.6058169603347778, 0.4366234242916107, -0.04863138124346733, -2.5537314414978027, 0.36670511960983276, -0.49177324771881104, 0.31379255652427673, -0.7180882096290588, -1.2411108016967773, 0.03969189152121544, 7.139552326407284e-05, -1.9373750686645508, -0.6246415376663208, -0.5241097211837769, -0.1279717981815338, 0.6569063067436218, 0.6620786786079407, 0.34393230080604553, -0.21895529329776764, -0.12991981208324432, 0.13977187871932983, 2.0811667442321777, 0.10266152024269104, -0.4077923893928528, 1.8374887704849243, -0.11711428314447403, -0.3139284551143646, 0.17567259073257446, 0.11133282631635666, -0.5468232035636902, 1.7111495733261108, -0.5744771361351013, 0.016405358910560608, 0.1831730455160141, 1.5023506879806519, 0.631684422492981, 0.0037327036261558533, -0.4288889169692993, 1.4369360208511353, -0.40945976972579956, 1.5142728090286255, -0.13837365806102753, -0.29145511984825134, -0.9831302762031555, 0.027562452480196953, -1.1885672807693481, -0.4057604968547821, 2.997098445892334, -0.09401413798332214, 0.9581027626991272, -0.028752153739333153, -0.22021673619747162, -0.8238463997840881, -1.6173356771469116, 3.5955421924591064, -0.41427940130233765, -0.013268209993839264, -0.04129498451948166, 2.3652145862579346, -0.1657029241323471, 0.17031928896903992, -3.023178815841675, -0.00848154816776514, -0.863516628742218, -0.09296012669801712, -0.2780969738960266, -0.08882426470518112, 0.01352300588041544, 0.6217278838157654, 0.8194102644920349, 0.9346137046813965, -1.0552726984024048, -0.1560501903295517, 0.12814292311668396, -0.8778504133224487, -0.8058182597160339, 0.2477709949016571, 0.05321015045046806, 0.25127702951431274, -0.19506579637527466, 0.07094769179821014, -1.2543423175811768, -0.6387925744056702, -0.06268820911645889, -0.3293892741203308, -0.1408870965242386, 0.3940935432910919, 0.02938705123960972, 0.12623260915279388, 0.011049846187233925, -0.012288426049053669, -0.923622727394104, 1.3088184595108032, 1.3278530836105347, -0.3697874844074249, 0.5204592347145081, 1.2090014219284058, 0.6386613249778748, 1.0692827701568604, 1.6717060804367065, -0.7044781446456909, 1.2458471059799194, 0.9476788640022278, 0.13770295679569244, 0.4250766336917877, -1.765248417854309, -0.09084384143352509, -0.012673569843173027, 0.6292129158973694, 1.2969608306884766, -0.9161304235458374, 0.0035818477626889944, 0.9520798325538635, 0.17836065590381622, -0.2990477979183197, -0.7250568866729736, -0.11886623501777649, -0.43378347158432007, 0.6919511556625366, 0.1253574937582016, 0.4210868775844574, 0.03322141245007515, -1.1328837871551514, -0.11450845748186111, 0.1863079071044922, 2.4252047538757324, -0.5536359548568726, -0.43569454550743103, 0.0004984576371498406, -0.14329779148101807, -1.092965841293335, -0.0012682239757850766, -0.0688687190413475, 1.3543307781219482, 0.015844330191612244, 1.5603383779525757, 0.41300979256629944, 0.02204815112054348, -0.00309195788577199, -0.08308758586645126, 0.4326547384262085, -0.11103516817092896, -0.22575031220912933, 1.0890092849731445, 1.1336493492126465, 0.833467960357666, 0.1620711088180542, -0.0728202536702156, 0.12622806429862976, -0.5971420407295227, -0.37562644481658936, 1.241978406906128, 0.21650850772857666, 0.618649423122406, 0.3563230335712433, -0.6496619582176208, 0.043276045471429825, -1.6912314891815186, 0.9553861618041992, -0.014544414356350899, -0.007329014595597982, 0.25334203243255615, -0.026911726221442223, -0.1289382427930832, -1.5133804082870483, 1.5400441884994507, -0.8165296316146851, -1.4621994495391846, 1.364030122756958, 0.6509894132614136, -1.6507374048233032, -0.1781611442565918, 0.07187478989362717, 0.7605040073394775, 0.5137891173362732, 0.8421286940574646, -0.012210452929139137, -0.09692628681659698, 0.2869490683078766, 0.16512079536914825, 0.27655279636383057, 0.19172075390815735, -0.262586385011673, 0.8412220478057861, 0.44386956095695496, 0.48583075404167175, 3.221637725830078, -0.5761831402778625, 1.2857502698898315, -0.37340137362480164, -1.159569263458252, 0.7177647948265076, -0.05492711439728737, -0.03539831191301346, -0.019296055659651756, 2.5851120948791504, -1.1814078092575073, -0.02069191075861454, 0.22860124707221985, -2.1788766384124756, -0.02865736000239849, -0.24594567716121674, 0.0844312235713005, 0.2839321494102478, 0.5315383076667786, 0.7667585015296936, 2.050787925720215, 0.37007978558540344, -0.1344783455133438, -0.14726322889328003, 0.513037919998169, -0.18581373989582062, -0.06719281524419785, -0.2530463933944702, 0.1738305538892746, -0.029613561928272247, 0.23420538008213043, -0.6062137484550476, 0.4065305292606354, 0.2830451726913452, 0.2064714878797531, -1.8705023527145386, 0.26535564661026, 0.7878788709640503, -1.1358858346939087, 0.21847213804721832, -0.08175774663686752, 0.6542779207229614, -0.7193924188613892, -1.7724554538726807, -0.019697124138474464, 0.8478521108627319, -0.445128858089447, -0.21444588899612427, 0.5584198236465454, -0.49596282839775085, 0.536384105682373, 0.03819594904780388, -0.5356601476669312, -2.019925594329834, -1.2038869857788086, 0.7564375996589661, -0.045448753982782364, 0.3315482437610626, 0.26461389660835266, 0.031117022037506104, 1.8867024183273315, -0.18300211429595947, -0.8781469464302063, 0.13952040672302246, 0.604009211063385, -0.8181862235069275, -0.32870087027549744, 0.1939174383878708, 0.02895999886095524, -0.27274078130722046, -1.8508692979812622, 0.14163656532764435, -0.4041110575199127, 2.6669907569885254, 0.26146045327186584, 1.3355376720428467, 0.4154055416584015, 0.16802604496479034, -2.506255865097046, 0.4935395419597626, -1.9347442388534546, -0.09547591954469681, 0.04603094235062599, 0.163692906498909, -0.22069241106510162, 0.043963897973299026, 0.83800208568573, 0.6054438948631287, -0.44632768630981445, 0.3428145945072174, 0.1275087296962738, 0.07481519132852554, 0.25329774618148804, 0.261405348777771, 1.0096558332443237, -1.151996374130249, -0.23257212340831757, 0.20729772746562958, -0.20272783935070038, -0.06657125055789948, 1.1312847137451172, -1.3170639276504517, 0.0884697213768959, 4.354765892028809, 0.20448361337184906, -1.5589888095855713, 0.008457615040242672, 0.16775454580783844, 1.1966885328292847, 0.4553990364074707, 0.06630054861307144, 0.012335672974586487, 0.6414062976837158, 0.9534911513328552, -0.22753499448299408, 0.2430507242679596, -0.5798195004463196, -2.7270281314849854, 0.056372206658124924, -0.10403524339199066, -0.913969874382019, -0.0771278515458107, -3.7964611053466797, -0.09722932428121567, 1.6052364110946655, 0.005138970445841551, -0.20518824458122253, 0.44009628891944885, -0.10176823288202286, 0.5315255522727966, 1.267153024673462, -0.27336034178733826, 0.03017193078994751, -0.013939330354332924, -0.4534010887145996, 0.3758181929588318, -0.6224300861358643, 0.86012202501297, 0.3303307592868805, -1.165001392364502, 0.7234492301940918, 0.47464945912361145, -2.1505422592163086, -0.04029221832752228, -0.09081509709358215, 1.3374077081680298, 0.3454783856868744, 3.1402413845062256, 0.9071069955825806, -0.0018616595771163702, -0.30615830421447754, 1.364018440246582, 0.19151471555233002, 0.007007479667663574, 0.45093730092048645, 0.0021230271086096764, 0.25629720091819763, 1.1891801357269287, -0.26299217343330383, -1.3825628757476807, -0.20192964375019073, -0.012956121005117893, -0.8705888390541077, 0.2113206386566162, 3.9638102054595947, -0.7543129920959473, -0.6363911628723145, 2.6992580890655518, -0.028899339959025383, -1.6467870473861694, -2.2214720249176025, -0.1811586171388626, 0.047890886664390564, -0.2197188436985016, 1.0456573963165283, 0.4290620684623718, -0.18394456803798676, 0.6433300375938416, 0.23717568814754486, -0.9696093201637268, -0.2398912012577057, -3.6303396224975586, -0.3884592056274414, 0.30058398842811584, -2.3138935565948486, -1.5776339769363403, -0.025342704728245735, 0.21839335560798645, 0.13341198861598969, 0.015189186669886112, -0.5138721466064453, 1.1174004077911377, -0.183442622423172, -0.4190463125705719, 1.130826473236084, 0.48206427693367004, -0.04140327870845795, 0.05255226790904999, 0.0869411751627922, -0.5273391008377075, -0.4246545732021332, 0.06969110667705536, -0.30536654591560364, 0.39010244607925415, -0.011519985273480415, -1.5386072397232056, -0.330219566822052, -2.837928295135498, -0.8742605447769165, 3.5973904132843018, -0.38803187012672424, -0.3279935121536255, -0.08201456069946289, 1.2428851127624512, 0.31079116463661194, -0.5517325401306152, 1.6847126483917236, 0.09437041729688644, -0.3355264961719513, -0.17582616209983826, -0.162223219871521, -0.349378764629364, -0.09301076084375381, 1.72867751121521, 0.8146106600761414, 0.6430426239967346, 0.3757527470588684, 1.431358814239502, 1.1311794519424438, 0.39182037115097046, 0.08514720946550369, 0.17127682268619537, -1.3850691318511963, -0.022923307493329048, -0.009458659216761589, 1.0435261726379395, -0.04448295384645462, 0.3445953130722046, -0.8880541324615479, -0.3763352632522583, -1.7867647409439087, -1.3843071460723877, -0.008107666857540607, 0.48260465264320374, 0.08876669406890869, 0.6150870323181152, 0.5426769852638245, 0.36693382263183594, 1.0039494037628174, 0.05010105296969414, 0.34566730260849, -0.21236249804496765, 0.4942489564418793, -0.6028850674629211, -1.6504660844802856, 0.3032762110233307, -0.1686667501926422, -0.11491868644952774, 1.1570484638214111, 0.10452272742986679, -0.2992282509803772, -0.10852321237325668, -0.27767205238342285, -0.05198958143591881, 0.16721974313259125, 0.2550239861011505, -0.768431544303894, 2.354717493057251, -0.01682138256728649, -0.942160427570343, -0.036803100258111954, -0.31499069929122925, -0.6997570395469666, 0.6178971529006958, -0.17848214507102966, -0.0018842376302927732, 2.5900931358337402, -0.046204689890146255, -0.08963687717914581, -0.746378481388092, -0.004641763400286436, -0.913918137550354, 0.28987249732017517, 0.5674366354942322, 0.03496846184134483, -0.6939912438392639, -0.0747893825173378, 0.31971198320388794, -0.33594900369644165, -0.09838688373565674, -1.1835471391677856, -0.10386507213115692, 1.499732255935669, 0.3789517879486084, -0.11881996691226959, 0.1595228612422943, -2.9143166542053223, 0.03508411720395088, -1.4982717037200928, 0.0327240452170372, -0.11011326313018799, 1.2049390077590942, -0.5049963593482971, 1.2498061656951904, 0.9449308514595032, -0.46649783849716187, 1.0953449010849, 0.055176328867673874, -0.004994528833776712, 0.5580316185951233, -0.3766029477119446, -0.37155699729919434, 0.5708359479904175, 0.09270068258047104, -0.1535714566707611, -0.48486220836639404, 0.008100801147520542, 1.51526939868927, -0.15114831924438477, 0.5732083320617676, 3.039076805114746, 1.825509786605835, 0.13372494280338287, -3.1656715869903564, -1.3430473804473877, 0.21745352447032928, -1.0224602222442627, -0.0275775995105505, -0.9665775299072266, 0.034771163016557693, -0.3293466866016388, -0.1597556620836258, 0.47517111897468567, 2.3032891750335693, -0.6743478178977966, -0.3986498713493347, -0.17418958246707916, -0.2273370772600174, 1.3092243671417236, -2.782545804977417, -0.7206932902336121, -1.4442168474197388, 0.34049513936042786, 0.15795814990997314, -0.032552000135183334, -0.016951126977801323, -0.5216678380966187, 0.5048460364341736, 0.5913479924201965, -0.37914541363716125, -0.13241106271743774, -0.6804057955741882, -0.03666626662015915, -0.5983203649520874, 0.2897986173629761, 0.935682475566864, -0.05190042033791542, 0.3827788233757019, 0.04022028297185898, -0.8014180660247803, -0.16966581344604492, -0.21434856951236725, 0.11963413655757904, -2.006179094314575, -0.12472868710756302, 0.3071884214878082, -0.15784743428230286, -0.31943053007125854, -1.2311444282531738, 0.2758638262748718, -0.025434091687202454, -0.06487646698951721, -0.5428141951560974, -0.19910021126270294, -2.772778034210205, -2.80206298828125, -1.4686496257781982, 0.7506443858146667, -0.030401065945625305, 1.3631173372268677, 0.39684101939201355, 0.26668673753738403, 0.814606785774231, -1.4555124044418335, -0.8426955342292786, 0.04260719195008278, -0.1193331927061081, 0.0507955439388752, -0.31869396567344666, -1.0295865535736084, 0.16004060208797455, -0.3427416980266571, -1.5716865062713623, -0.41171059012413025, 0.03334343805909157, -0.39160311222076416, 0.2772068381309509, -0.30852147936820984, 0.1697148084640503, -0.05080337077379227, -7.56844110583188e-06, 2.482342004776001, -0.10531890392303467, -0.24501800537109375, 1.2157319784164429, 0.6161932349205017, -0.2909512221813202, -0.7398513555526733, -2.0438573360443115, -0.05803802236914635, 3.30478572845459, -1.3649564981460571, -0.6844666600227356, -0.040912069380283356, 0.9081048369407654, -0.21148721873760223, 0.005928267724812031, -0.39892643690109253, 1.2402626276016235, 0.01484216470271349, 1.9923105239868164, -2.440599203109741, -0.010382212698459625, -0.8088918328285217, -0.07946819067001343, 0.4764288663864136, 1.2999857664108276, 0.5531569123268127, 0.19261805713176727, -0.6223885416984558, -0.27994629740715027, -0.3335973620414734, -0.11400938034057617, 0.03435879945755005, 0.14596721529960632, -0.2350281924009323, 0.37796351313591003, 0.8650433421134949, -0.0465693324804306, 2.085193395614624, -0.06526859104633331, -0.4317847788333893, 1.077298879623413, 0.9853057265281677, -0.0699780061841011, 0.6972389221191406, -0.10566497594118118, -1.1099114418029785, -0.28939807415008545, -1.8938096761703491, 1.4613487720489502, 1.745885968208313, 0.1966930776834488, -0.43505141139030457, 0.07659315317869186, 0.19278211891651154, 0.12330391258001328, 0.03182118013501167, 0.05357765406370163, -2.1105685234069824, 0.8037482500076294, -0.2674802839756012, -1.458422303199768, -0.1575426161289215, -1.4832520484924316, 0.08529277890920639, 0.29583773016929626, -0.03059684857726097, 0.7481412291526794, 0.06195114552974701, -0.4004597067832947, -0.5919649600982666, -0.3231009244918823, -1.0429699420928955, -0.6975131034851074, -0.961532473564148, -0.5040922164916992, -0.759981095790863, -0.09416213631629944, -0.0025307228788733482, -0.16990745067596436, -0.8675463199615479, 0.22996774315834045, 0.6148169040679932, -0.09401408582925797, 0.20759528875350952, 0.023547278717160225, 0.9612759947776794, 0.04273414611816406, -0.013974565081298351, -0.08028078079223633, 0.13665321469306946, 0.9829233288764954, 0.42718347907066345, 1.907342791557312, 0.8734767436981201, -1.1054151058197021, -0.9940270781517029, -0.04735876992344856, 0.6279558539390564, 0.20869626104831696, 1.0487689971923828, -0.2718197703361511, 0.38391387462615967, 0.41158542037010193, -0.19509854912757874, 0.814914345741272, -0.1422189325094223, -3.3679449558258057, -0.3069586753845215, 0.23542509973049164, 0.15635381639003754, 0.37866443395614624, -1.4181522130966187, -0.07423213124275208, 0.397174209356308, 1.0852292776107788, -0.28538990020751953, -0.09120341390371323, 1.0046076774597168, 1.0869861841201782, -0.6071984767913818, -0.2077447474002838, -0.35040411353111267, -0.12858623266220093, 0.791234016418457, -1.3975452184677124, 0.25757962465286255, 0.49102839827537537, 0.41951799392700195, -0.1595536321401596, 0.5225531458854675, -0.3585493564605713, -2.13161563873291, 0.3179375231266022, -0.14419040083885193, 0.3283575177192688, 0.015442577190697193, -0.23038361966609955, -0.28604409098625183, -0.09710186719894409, 1.1503139734268188, 0.16271516680717468, 0.007777586113661528, 1.2414013147354126, -0.08584237843751907, -0.19738228619098663, -0.02213815040886402, -1.7536420822143555, -0.8701809048652649, 0.06763209402561188, 2.3118085861206055, -0.7227831482887268, -0.08993718773126602, -0.11404138803482056, -0.00037984843947924674, 0.45723092555999756, -0.006410511676222086, 3.3811428546905518, 0.9357009530067444, 0.08696858584880829, 0.7600181102752686, 0.1263941377401352, -0.4766228199005127, 0.16640473902225494, 0.09253091365098953, 0.8483052253723145, -0.4691377282142639, 0.6545324921607971, -0.18016605079174042, 0.027465639635920525, -0.24629220366477966, 0.9661028981208801, -1.5634196996688843, -0.793242335319519, 0.14239825308322906, 0.05492774024605751, -0.5362004637718201, 1.228325605392456, 0.16101321578025818, -0.10217314213514328, 0.10071855783462524, 0.11589548736810684, 0.3802357316017151, 1.8642367124557495, -0.062405966222286224, -0.11378578841686249, 0.30773064494132996, -0.47568994760513306, -0.2406741976737976, 0.16429908573627472, -1.5354899168014526, 1.1148568391799927, -1.17840576171875, -0.09821115434169769, -1.3195680379867554, -0.1742708832025528, -0.08448942750692368, 0.1794675588607788, 0.07425980269908905, 0.007515530567616224, 0.07028624415397644, -1.4503086805343628, 0.20358529686927795], [0.5696208477020264, 0.8813925981521606, 0.27340492606163025, 0.8867956399917603, 0.6597862839698792, 2.335170030593872, -0.018343554809689522, -0.9533454775810242, 2.03164005279541, 2.5390381813049316, -3.022963047027588, 1.0963788032531738, -0.30371496081352234, 0.33672472834587097, -0.30323973298072815, -0.26843076944351196, 0.39590582251548767, -0.21244193613529205, 0.17837189137935638, 1.6734648942947388, 2.218876838684082, 0.5630599856376648, 0.020010648295283318, -1.1467348337173462, -0.22168239951133728, 0.42854422330856323, 0.1278151422739029, -0.570275604724884, 0.30519336462020874, 0.033217038959264755, -1.2810865640640259, 1.8216427564620972, -0.26005062460899353, -0.42341333627700806, 0.07029231637716293, 1.9142757654190063, -0.5065426826477051, -0.24086080491542816, 0.35021522641181946, 0.600959300994873, -1.1089941263198853, 0.016333887353539467, 0.025511227548122406, 0.31337010860443115, -0.0842638686299324, 0.24260984361171722, 0.09122534096240997, 1.0377882719039917, 1.9184683561325073, -0.2645459771156311, -1.146437644958496, 0.37752822041511536, -0.017133766785264015, 0.44114071130752563, 0.12542003393173218, 0.18665942549705505, -0.2926870584487915, -0.007727215066552162, 0.01657985895872116, -0.03719039261341095, 0.18974071741104126, -0.5340636968612671, -0.10439594835042953, -1.692679762840271, 1.8026028871536255, 0.18317489326000214, 1.0117131471633911, -0.31228649616241455, 0.0231856107711792, 1.4835798740386963, -0.09992705285549164, -0.6440079808235168, 0.2844977378845215, -0.18836748600006104, 0.09224333614110947, 0.6455222964286804, -0.4126966893672943, 2.193840265274048, -0.023848583921790123, 0.16562992334365845, -0.46185624599456787, -0.3964903950691223, 0.07049258798360825, 0.029669681563973427, -0.02751229703426361, 0.6817253232002258, 0.23319286108016968, 0.19478698074817657, 0.908078134059906, -0.4738302230834961, -1.4908252954483032, 0.18936927616596222, -0.5110411643981934, 0.18359346687793732, 0.551961362361908, -2.576899528503418, 0.1191752627491951, -0.9733631014823914, 0.0833418145775795, 0.005530767608433962, -0.12435854226350784, -0.5842844843864441, -0.508786141872406, 0.03275458887219429, 0.10129720717668533, 0.016254503279924393, -0.01038410048931837, -0.23326146602630615, -4.631569862365723, 1.2873469591140747, -0.69981849193573, -0.022277744486927986, 0.4293998181819916, 0.17515000700950623, 0.5278708338737488, -0.15298816561698914, 0.24919553101062775, -0.0960170179605484, -0.2382580041885376, 0.5054978132247925, -0.0652984082698822, -0.9411020874977112, -0.08688487112522125, 1.2631323337554932, 0.02326546050608158, 1.352785587310791, -0.8879645466804504, 0.061178743839263916, 0.19978190958499908, 0.45020490884780884, 1.111851453781128, -4.405479907989502, 0.1082993671298027, 0.19218865036964417, 0.26184263825416565, 0.675514817237854, 0.09269353747367859, 0.7841814756393433, -0.2695336937904358, 0.31982335448265076, 0.43451374769210815, -0.023768436163663864, 0.17644543945789337, -0.006186843384057283, -2.8920629024505615, 0.3833337128162384, 0.4052002727985382, 0.8144568800926208, -0.0901392474770546, -0.08962423354387283, 0.5734783411026001, -0.43885868787765503, -0.04837300628423691, -0.023620612919330597, -3.6996777057647705, -1.5871797800064087, -1.1588687896728516, -0.16075047850608826, -0.24724741280078888, 2.0610275268554688, -0.000139579686219804, 0.007731679826974869, 0.12044176459312439, -0.732098400592804, -0.8106361627578735, -0.05179322510957718, 0.4091437757015228, 0.1594390869140625, -0.6186479926109314, -0.007672172971069813, 0.0473293736577034, 1.529135823249817, -0.8351908922195435, 0.027627309784293175, -0.1629953384399414, -0.13674017786979675, -0.11418020725250244, -0.11919844150543213, 1.147314429283142, 2.2768728733062744, -0.6636418104171753, 0.035815056413412094, -0.03483056277036667, -0.12036589533090591, -1.5776797533035278, 1.7360702753067017, 0.30170828104019165, 0.011767711490392685, 1.7451646327972412, 2.9685657024383545, 0.18201366066932678, -0.9373432397842407, -0.023776501417160034, 0.8275269865989685, -0.35527995228767395, -0.4020068347454071, 1.8588026762008667, -0.24040812253952026, -3.4402573108673096, 2.569990634918213, 0.3619334399700165, 0.016937043517827988, -0.8058424592018127, -0.06047143042087555, -0.08986080437898636, 0.052891168743371964, 1.4451954364776611, 0.41526973247528076, 2.201430320739746, 0.19879569113254547, -0.07242299616336823, 0.08616233617067337, -0.07341139763593674, 0.0344027504324913, -0.22272977232933044, 0.8994849920272827, 0.45345553755760193, -0.32861390709877014, 0.08456747978925705, -0.2043987512588501, -0.4223566949367523, 0.5700238347053528, -0.12247920781373978, 1.1768995523452759, -0.5828872323036194, 0.3065599501132965, 1.5830397605895996, 0.3470211327075958, 1.8022408485412598, 0.0923844501376152, 0.12873606383800507, 0.9692050218582153, -1.063976526260376, 1.1443804502487183, -0.6779805421829224, -1.6526613235473633, 0.018352337181568146, -0.2641008496284485, -0.15534371137619019, -0.014763670042157173, 0.5621612071990967, 0.517219066619873, -0.24231481552124023, -0.7097573280334473, 0.3223801553249359, 0.13690389692783356, -2.555971622467041, 0.997369647026062, 0.25781887769699097, 0.10067813843488693, 0.07215756922960281, -0.35602685809135437, -1.1844578981399536, -0.752418041229248, 1.0659655332565308, 0.03027060069143772, -0.4382091462612152, 0.10604682564735413, -0.015152045525610447, 0.6854234337806702, 1.5499718189239502, 1.478371262550354, 0.3270989656448364, 1.6037777662277222, -0.11359139531850815, 0.1246720552444458, -2.339914083480835, -0.015380000695586205, 0.49411576986312866, -1.4645483493804932, -0.007088230457156897, -0.9733599424362183, 0.29321080446243286, 0.24041365087032318, 0.2916758060455322, -0.22792616486549377, -0.5224125981330872, 0.3626720905303955, -0.55925053358078, 0.32127445936203003, -0.02356462925672531, 1.2833467721939087, -0.8507694602012634, 0.10447952896356583, 0.6152641773223877, 0.9323998093605042, 0.8601545095443726, 0.2138143628835678, -0.36638185381889343, 0.3432522118091583, -0.12986886501312256, -0.21347598731517792, -0.456016480922699, 2.7766940593719482, 0.26899072527885437, -0.009080885909497738, 0.3928297460079193, -2.146212100982666, -0.4170343577861786, -0.09371498972177505, -1.1074182987213135, -0.0057393694296479225, 0.20027495920658112, 0.44319063425064087, 0.9027053713798523, -0.8757600784301758, -0.05280563235282898, 2.780026435852051, 1.771531581878662, -0.22742943465709686, 0.027209442108869553, -0.7761077284812927, 0.0615026131272316, -1.2826381921768188, 0.1308354139328003, -0.1775827556848526, -1.3612139225006104, -0.2095775455236435, 2.2884371280670166, -0.12585040926933289, 0.02847471460700035, -1.4457778930664062, -0.13937854766845703, 0.5815834999084473, -0.1355506330728531, -0.6015468239784241, -0.896217942237854, -2.005659818649292, -0.19628842175006866, -0.5975383520126343, -0.0986398234963417, -2.1540863513946533, 0.01282896101474762, 0.02345779724419117, 0.4298134744167328, -1.508950114250183, 0.23176686465740204, 0.4900122880935669, 0.3021792769432068, 0.8504962921142578, 0.13656766712665558, -1.4774473905563354, -0.23763154447078705, -0.08424306660890579, 0.23819632828235626, -1.0009005069732666, 0.24598784744739532, -0.03151792660355568, -0.3995113968849182, -0.14007897675037384, 2.5158884525299072, 0.38270303606987, -1.1758568286895752, -0.0280239749699831, -0.3986820578575134, 0.7163565158843994, 0.013986985199153423, -0.8487526774406433, 4.261942386627197, 0.12172127515077591, -0.08199648559093475, -0.6919474601745605, -2.053823947906494, 0.09841378778219223, -1.5448299646377563, -1.159101128578186, 0.10624633729457855, -0.010988251306116581, -1.0199337005615234, 2.4315946102142334, -0.02564084902405739, 0.14372429251670837, 0.3192891776561737, -1.0099925994873047, 0.530272901058197, 0.24556119740009308, -0.8711835741996765, 0.02308652177453041, -0.525542676448822, 0.20679517090320587, 0.8909419775009155, 1.9425623416900635, 0.029060501605272293, 1.249859094619751, 1.8326390981674194, -0.2947981357574463, 0.014064792543649673, 0.042476605623960495, -0.49407947063446045, 0.22857563197612762, -0.43878984451293945, 0.02424958162009716, -0.023675499483942986, 0.021811069920659065, 0.44374963641166687, -0.005906083155423403, -0.00014845172700006515, -0.04935917630791664, 0.3205746114253998, 0.00014508984168060124, 0.5294133424758911, -0.2805907428264618, 0.2515545189380646, 0.8008959889411926, 0.06525671482086182, 0.3481244444847107, -1.7372108697891235, -0.26193466782569885, -2.4941558837890625, -0.09846242517232895, -0.3672352135181427, -2.9810848236083984, 0.30990782380104065, 1.0121251344680786, 0.017295582219958305, 0.30607083439826965, 0.538733720779419, 1.14264714717865, -0.30340245366096497, -0.6968958377838135, -0.360884428024292, 0.6677164435386658, -0.09049677103757858, 1.2623026371002197, -1.8611412048339844, -2.30025053024292, 0.04947743937373161, 0.9750581979751587, -0.7280060052871704, 0.13448256254196167, -0.19449403882026672, 0.1950879842042923, -0.38601893186569214, -1.4988759756088257, -1.9306097030639648, 0.9959753751754761, 0.05624406039714813, 0.10336274653673172, 0.002312827156856656, 0.28192663192749023, -0.2145269215106964, -0.7124512791633606, -0.2971227467060089, -0.14672408998012543, 0.6006011366844177, -1.0519853830337524, -0.23606084287166595, -3.438936233520508, -1.448720932006836, -0.008235638961195946, 0.11792498826980591, -1.146138072013855, -0.0032628353219479322, 0.13258272409439087, 2.377371072769165, -0.6047409772872925, -0.13644136488437653, -1.2125825881958008, -0.19245804846286774, 0.01693972572684288, -0.10916575789451599, -0.2749113142490387, 0.008153765462338924, 0.1319255530834198, 0.07275673747062683, 0.15865729749202728, -1.2828880548477173, -0.05930845066905022, -0.09571776539087296, -0.0927605852484703, -0.12525834143161774, -1.2606658935546875, -0.03907470405101776, -1.8421205282211304, -0.4596520960330963, 0.6779199242591858, 0.8381030559539795, 2.7717666625976562, 0.005470301490277052, -2.463261842727661, -0.16305866837501526, -0.03094778023660183, 0.4993577003479004, 0.90630704164505, -0.7075641751289368, 1.297013521194458, -0.7243317365646362, 0.2245461493730545, -0.9902121424674988, -0.008528269827365875, 0.6425144672393799, 0.3528519868850708, -2.0995097160339355, 1.3968172073364258, 0.6829106211662292, 0.20469930768013, -0.3079475164413452, 0.13973648846149445, 2.6085264682769775, 1.5930538177490234, 3.1999895572662354, 1.6502580642700195, -0.9195072054862976, 0.055794913321733475, 0.14909499883651733, 0.004615446086972952, 0.08394160866737366, -0.014847267419099808, -0.3497183918952942, 1.1061713695526123, -2.5974061489105225, -0.0003413353697396815, -0.0741834044456482, -0.9655851721763611, 0.18727967143058777, -0.2568137049674988, 2.5211074352264404, -0.40809205174446106, 1.2459067106246948, 1.4445836544036865, 0.38917043805122375, 0.09061544388532639, 0.056137923151254654, 0.9461740851402283, 0.32343968749046326, 0.1559760570526123, -0.904732346534729, 0.3186393678188324, -0.039366863667964935, -1.983284831047058, -0.40329498052597046, -0.8539403676986694, -0.08916175365447998, 0.28822392225265503, 1.2359895706176758, -0.6063022017478943, -0.8749294877052307, 0.6009203195571899, -3.118671417236328, 0.28464967012405396, -0.9791741371154785, -0.0003499043523333967, 4.749974727630615, -0.008422925136983395, -0.04036986827850342, -9.79094547801651e-05, -0.5488290786743164, 0.3763202428817749, 0.012590121477842331, -0.011912711896002293, -0.07239288836717606, -0.07973367720842361, -0.569417417049408, 0.05493869632482529, -1.1046340465545654, 0.549078106880188, -0.9647989273071289, 0.3503309488296509, -0.14482973515987396, 0.0516454316675663, -1.2628278732299805, -0.9334441423416138, -0.08301379531621933, 0.1210816353559494, -2.5266637802124023, 0.5852797627449036, -0.029632648453116417, -0.35078588128089905, -0.5493561625480652, -0.6400763988494873, -0.4999276101589203, -5.028891563415527, 1.2673847675323486, 1.0775882005691528, 0.3589373230934143, 0.5945004224777222, -3.119809150695801, 0.6841830611228943, 2.243795394897461, 0.3065163195133209, 0.338800847530365, 0.4411601126194, -0.1029859185218811, -0.23070628941059113, -0.0665128082036972, -1.9341953992843628, 0.12257980555295944, -0.2221120446920395, 0.26347795128822327, -0.07695725560188293, -0.36579224467277527, -0.16179688274860382, 0.026496263220906258, 0.11894474923610687, 0.004853794351220131, 0.009782749228179455, -1.3616136312484741, 1.012999415397644, 0.3066422641277313, 0.145485520362854, 0.7481591105461121, -0.18839389085769653, -0.118619404733181, 1.0636740922927856, 0.19766069948673248, 0.6426151394844055, 2.1439285278320312, -2.1813323497772217, 0.6922402381896973, -1.5668282508850098, -2.2170512676239014, -0.559931755065918, -1.3930007219314575, -0.7051768898963928, -0.13407932221889496, 0.8617523312568665, -0.15942133963108063, 0.0699349045753479, 0.08119356632232666, -2.18814754486084, 0.007187846582382917, 0.003921301104128361, 0.33793458342552185, -0.3786121904850006, -0.22497566044330597, 0.22045083343982697, 2.754134178161621, 0.21026752889156342, 0.09421811252832413, 0.6261957287788391, -0.6329665184020996, -0.23843833804130554, -0.005543138366192579, -0.1719174087047577, 1.2163959741592407, 0.07208626717329025, -1.2115658521652222, 0.0415353886783123, 0.07137782126665115, 0.47125309705734253, -0.46923670172691345, 0.04838065803050995, -0.040139392018318176, -0.4487692415714264, -0.060422904789447784, 0.5330497622489929, -1.1842091083526611, 0.04554666951298714, 1.4654614925384521, -0.05799321457743645, -0.1190260723233223, 0.5453371405601501, -0.17974145710468292, -0.20462654531002045, 0.05945761501789093, 0.08316472917795181, 0.019384263083338737, 0.15086965262889862, 0.6893475651741028, 1.0632522106170654, 0.18652787804603577, 0.5240844488143921, 0.040728937834501266, 0.008439048193395138, -1.4872667789459229, 0.06605193763971329, -2.4144768714904785, 0.009537763893604279, 0.20585307478904724, -0.0773375928401947, -0.4838048815727234, 0.24436718225479126, -0.03351602330803871, 0.9499126076698303, -2.481128692626953, -3.2346720695495605, 0.9656259417533875, 0.3829212486743927, -1.2779134511947632, -0.02162718027830124, -0.06612735241651535, -0.106527179479599, -0.5631468296051025, 0.4000597596168518, -0.10881023108959198, 0.03341156616806984, 0.07478017359972, 0.20509971678256989, -2.5125558376312256, -0.08751232177019119, 3.123875379562378, 0.8852618932723999, -0.22783386707305908, -0.527086079120636, -0.16864176094532013, -2.6206889152526855, -0.3479539752006531, -0.06474723666906357, 0.47568491101264954, -2.9101555347442627, -0.0036467446479946375, -2.5293712615966797, 0.11495493352413177, -0.08868629485368729, 0.6780130863189697, 0.2558744549751282, -0.16849635541439056, 0.16679006814956665, -0.3388669192790985, 0.9645203948020935, 0.9948656558990479, -1.9784832000732422, -1.30912446975708, 0.968647837638855, 0.041707538068294525, 0.54873126745224, 0.009101307019591331, -0.14073306322097778, 1.5701733827590942, 0.025646647438406944, -0.27415961027145386, 0.030064424499869347, -3.8564441204071045, -0.3642185926437378, 0.3338460326194763, -1.4415278434753418, -0.07353650778532028, 0.27192825078964233, 0.4908044934272766, -0.02863992191851139, -0.9874287247657776, 0.09129603207111359, -0.7113324999809265, -0.2296406477689743, 0.7166476249694824, -0.09692031890153885, -0.04076479375362396, -1.418495535850525, -0.06529225409030914, 1.6330591440200806, 0.001024747034534812, -0.9340381622314453, 0.6747289896011353, 0.44720426201820374, 0.2026139497756958, -0.7991046905517578, 0.0017314426368102431, -0.13124419748783112, 0.7478106021881104, -0.6322153210639954, -0.372026652097702, 0.18921662867069244, 7.452785015106201, 0.7002307176589966, -0.5984617471694946, -0.022301100194454193, 0.006354313809424639, 0.10047846287488937, 0.2241065800189972, -0.021412156522274017, -0.25592294335365295, 0.04559639096260071, 0.18863283097743988, 0.3977557420730591, -0.22858375310897827, 1.3813766241073608, -0.21958620846271515, -0.28884580731391907, 0.021323183551430702, -0.9113190770149231, -0.02456061914563179, -2.6444976329803467, 0.35052457451820374, -0.0004695869574788958, -0.012703140266239643, -0.14306867122650146, -0.7217352986335754, 0.29687735438346863, 0.10408754646778107, -3.4033117294311523, -0.005223702639341354, -3.311323881149292, 0.08450651168823242, -0.09849772602319717, 0.12044964730739594, 0.019586917012929916, -1.1869771480560303, 1.700300931930542, 0.6930370926856995, 0.48419708013534546, 0.2330562025308609, -0.6706711649894714, 0.08217722177505493, -0.14321450889110565, -0.0010027666576206684, 0.08370356261730194, 0.048310600221157074, -0.24369898438453674, -0.036367930471897125, -0.05743168666958809, 2.3993985652923584, -0.05402547866106033, -0.5023698210716248, -0.038174912333488464, -2.7539145946502686, 1.0186563730239868, -0.500390887260437, -0.32149651646614075, 0.03438740596175194, -1.084896445274353, -0.2261762171983719, 0.015150153078138828, -0.5520137548446655, 0.048471495509147644, -0.04877402260899544, 0.045773111283779144, -0.2551630139350891, 0.10495618730783463, -0.4632469713687897, 0.8510714173316956, 0.3811759054660797, 0.25181999802589417, 0.2462160289287567, -0.28262969851493835, 0.8179253935813904, -0.7083966135978699, 0.007568931672722101, 0.2772417366504669, 0.12878799438476562, -0.2411147952079773, 0.24434228241443634, 0.04172930866479874, -0.6598013639450073, -0.4489021897315979, 2.49505877494812, 1.2289963960647583, 0.6185743808746338, 1.0067989826202393, -0.7469418048858643, 0.16073033213615417, 0.7770305275917053, -0.4372158646583557, -0.8008100390434265, -0.8055907487869263, -0.5714510679244995, -0.4837454557418823, -1.8779178857803345, -0.0710296630859375, -0.041876617819070816, -0.12092279642820358, -0.09176456183195114, -0.35099947452545166, -0.017224768176674843, -0.006013429258018732, -0.1970367133617401, -0.07311917841434479, 0.17610754072666168, 1.7095915079116821, -0.0061151133850216866, 2.4470458030700684, -0.2616061270236969, -0.23870158195495605, 0.05020947381854057, 0.06285196542739868, -0.3799426257610321, 0.03794233873486519, -0.6108673214912415, -0.22563162446022034, 0.297846257686615, -0.6524943709373474, -0.6621623635292053, 0.4986796975135803, -0.20269246399402618, 0.0058731394819915295, 0.10619018971920013, -0.7651121616363525, 0.0006481012678705156, 0.482315331697464, 0.5117114186286926, 0.6019077897071838, 0.07126012444496155, -0.36402758955955505, -0.9197379946708679, -0.3035804331302643, -0.9509986042976379, -0.073834128677845, 0.03635189309716225, -0.1332574486732483, 0.2325008064508438, -1.5317667722702026, 0.0878402441740036, 0.7296608686447144, -0.008968762122094631, 0.46777188777923584, -1.4373102188110352, 1.1505542993545532, 1.3781447410583496, 0.11677908152341843, -0.3746751546859741, 0.7943171262741089, -0.3940470218658447, -0.38240721821784973, 0.34698042273521423, -0.016623755916953087, 0.5175272226333618, 0.07560507953166962, 0.21525844931602478, -1.0605794191360474, 0.9395374059677124, 0.06064559146761894, 0.64736407995224, -2.268296241760254, 0.987928032875061, 0.012189628556370735, 0.04787737876176834, 0.2495688796043396, 1.7193907499313354, 1.4834377765655518, -0.25180038809776306, 0.7410008907318115, -0.02256324701011181, 0.435649037361145, 0.16788198053836823, 0.26960518956184387, 1.1708195209503174, -0.314846932888031, -1.4020087718963623, 0.11079619079828262, -0.13739456236362457, 0.5674794316291809, -0.04238009825348854, -0.8658053874969482, 0.34413182735443115, -0.6053080558776855, -0.11079176515340805, 0.4697795510292053, 0.44465887546539307, -0.6764008402824402, -1.3781991004943848, -0.3302132189273834, -0.7478572130203247, 1.2146503925323486, 0.23521475493907928, -1.4219762086868286, 0.07226265221834183, 0.0349838063120842, 0.24370929598808289, -3.9451167583465576, 0.1970837116241455, 0.4642193913459778, -0.9156541228294373, 0.125874862074852, 0.31209442019462585, -0.03626322001218796, -2.151834487915039, -0.24949021637439728, 1.4665945768356323, 1.7633147239685059, -0.09539441764354706, 0.12129440158605576, -6.352294921875, 0.5005102753639221, -4.508913993835449, -0.41230812668800354, 0.7488669157028198, 0.7344106435775757, 3.355393648147583, -0.010197736322879791, 0.1592942774295807, 0.43784067034721375, 0.5169353485107422, -0.0973280519247055, -1.3540410995483398, 0.026533739641308784, -0.054170940071344376, -0.16015389561653137, 0.5538679361343384, 0.4670460522174835, -1.6053367853164673, 0.022969258949160576, 0.30130550265312195, 0.0975225418806076, 0.38613465428352356, 0.040473587810993195, -0.5755142569541931, 0.4822056293487549, -0.14722368121147156, -0.008951258845627308, -0.034041307866573334, 1.404834270477295, 0.5500741004943848, 1.0567758083343506, 0.07977671176195145, -3.1174073219299316, -0.012491371482610703, 0.37561991810798645, 0.3883560001850128, -2.2128348350524902, 0.053414590656757355, -0.03687407076358795, 1.6155418157577515, 1.6014195680618286, -0.8745472431182861, -0.3251834511756897, -0.21015240252017975, 0.1354771852493286, -0.04349713772535324, -0.29103001952171326, -0.025805320590734482, -2.064850091934204, -0.0699683129787445, -0.6971774697303772, 2.3976001739501953, -2.8427469730377197, 1.3798110485076904, -0.15435516834259033, -0.21268726885318756, -1.121928095817566, -0.9796536564826965, -0.6234534978866577, -2.7177200317382812, 3.0468599796295166, -1.1486014127731323, -0.0029433080926537514, -0.09608620405197144, -0.011213699355721474, -0.6307171583175659, -0.06759536266326904, 0.25646287202835083, -0.16407079994678497, 0.46650174260139465, -0.1660633683204651, 0.02756086364388466, 0.08363892138004303, 0.060392316430807114, 1.4860419034957886, -0.1512293964624405, -1.3159018754959106, 0.33114704489707947, -0.009297673590481281, 0.0645718201994896, 0.1449228823184967, -0.021766606718301773, 0.8125666379928589, 0.033114489167928696, 0.8178521990776062, -0.9138950109481812, 0.0059070405550301075, 0.9183198809623718, -0.7729821801185608, -0.14339807629585266, 0.40012645721435547, -0.35104480385780334, -1.807055115699768, 0.22099950909614563, 1.1595563888549805, -0.04809478297829628, -0.022018130868673325, 0.03470033034682274, 0.5620004534721375, 1.8664278984069824, -0.07917694002389908, 1.722408652305603, -0.12410220503807068, -0.1762455552816391, 0.1912183314561844, 0.10854065418243408, -0.16198191046714783, 1.0167337656021118, -0.02323099784553051, -0.14696748554706573, -0.6212472915649414, 0.9282777905464172, -0.8712000846862793, -0.01458391360938549, -0.04989800229668617, -3.673444986343384, 0.13733330368995667, 0.15797723829746246, 0.7382116317749023, -3.9780666828155518, 1.294267177581787, -0.464110404253006, -0.05174776166677475, 0.22601225972175598, 0.25080159306526184, -0.5571132302284241, -0.03594900667667389, -0.5622648000717163, 0.48271623253822327, 2.5845704078674316, -0.8353947997093201, 0.22863726317882538, -0.1516542136669159, 0.011665835045278072, 0.09423269331455231, -1.1640478372573853, 1.4540694952011108, 0.0034905290231108665, 0.15938732028007507, 0.955315113067627, 0.17474576830863953, -0.14502514898777008, 0.7124854922294617, -0.2860446274280548, -0.10474744439125061, 0.38694876432418823, -0.028391143307089806, 0.818483293056488, 0.8168573975563049, 1.6154639720916748, 0.12427766621112823, 0.11203259229660034, -0.6960119009017944, -0.11601226031780243, -2.123934268951416, -0.08132145553827286, 1.3930975198745728, -0.1808289885520935, 0.8946030139923096, 0.038768090307712555, -0.13077068328857422, -1.7792515754699707, -0.1472884565591812, 1.007441520690918, -0.8307554125785828, 0.45085614919662476, 0.035799041390419006, 1.022661805152893, -0.7166678309440613, 0.022676313295960426, 0.00994347408413887, 0.11668139696121216, 0.07989716529846191, 0.38592806458473206, 0.1511979103088379, 0.36561864614486694, 0.22640012204647064, 0.027161933481693268, 0.1428033411502838, -1.1703476905822754, -0.8711016774177551, -0.0010787644423544407, 0.10061725974082947, 0.17830701172351837, 0.11775597184896469, 0.01535249873995781, 0.038733966648578644, -0.490883469581604, 0.8450236916542053, -0.28631147742271423, 0.12049718201160431, -0.3108632266521454, 0.1992771327495575, 0.8352190852165222, 0.41003668308258057, 0.5504587888717651, 0.18235541880130768, -0.7040691375732422, 0.37387222051620483, -0.42281508445739746, -2.096018075942993, 2.9851651191711426, 0.21010859310626984, 0.47259828448295593, -0.819550096988678, -1.061799168586731, -0.7376161217689514, 0.17979858815670013, -0.020236371085047722, -0.10667938739061356, 0.4557099938392639, 1.1693814992904663, -0.043664030730724335, -0.7301042079925537, 0.3979487419128418, -0.3427508473396301, 0.03566786274313927, -0.4885209798812866, -0.17662835121154785, -0.23126362264156342, 0.14684796333312988, 0.9532527923583984, 0.7486793994903564, -0.048281196504831314, 1.7936983108520508, -1.1027002334594727, 1.1049407720565796, 0.00179740646854043, -0.09832189232110977, -0.25314629077911377, 0.07427913695573807, 3.8208167552948, 0.03900524228811264, 0.17089414596557617, -1.6475332975387573, -0.4080891013145447, -0.4922536313533783, -1.5772918462753296, 0.0539691299200058, -0.12881571054458618, 3.026533603668213, 2.7800583839416504, 0.6065099239349365, 0.10783591121435165, 0.2321505844593048, -0.9003810286521912, 0.1400485783815384, -0.12158799171447754, -0.015113313682377338, 0.0005936822271905839, 0.1865159273147583, 1.421359896659851, -0.6350336670875549, -0.02947942726314068, -0.14357787370681763, 0.3795844316482544, 0.1787131130695343, 1.354253888130188, -0.5639882683753967, -0.004422448575496674, -0.48183029890060425, 0.25018635392189026, -0.35154736042022705, -0.21740426123142242, -1.4751300811767578, 0.9572616815567017, 0.0929175317287445, -0.6028466820716858, 0.16627532243728638, -0.3615519106388092, 0.6287136077880859, 0.3230758011341095, 3.8997409343719482, -0.06192388758063316, 0.927012026309967, -1.0644004344940186, 0.1561659276485443, 0.23781254887580872, 0.7362927198410034, -2.9462974071502686, 1.0256060361862183, 0.43101951479911804, 0.254425585269928, -1.767601728439331, -0.021942876279354095, -0.5628106594085693, 0.7235403656959534, -0.46014395356178284, -0.633037269115448, 0.7217270135879517, -1.3175572156906128, 0.809061586856842, -1.6976877450942993, -1.770066261291504, -1.3205574750900269, 0.00910087302327156, -0.3692139983177185, 0.37837642431259155, 0.21780920028686523, -0.04713832959532738, -0.08517032116651535, -4.018670558929443, 0.012363104149699211, -0.7143646478652954, 0.14165712893009186, 0.305955171585083, -0.17892129719257355, -0.6219606995582581, -0.08381038159132004, 0.1042853593826294, 1.9782359600067139, -0.22409722208976746, -0.39670851826667786, 0.32556259632110596, -1.0019901990890503, 0.3481353223323822, 0.07349182665348053, -1.9869412183761597, -0.5649109482765198, 1.6290196180343628, -0.09779050946235657, -0.6090345978736877, 0.10417404770851135, 0.0065946439281105995, -0.8534513711929321, -0.49799853563308716, 0.70020991563797, 0.17692457139492035, -1.1659952402114868, -0.6092832684516907, -0.10090696811676025, -0.1951184719800949, -0.057922519743442535, 0.010134540498256683, -0.11309438198804855, -0.24751709401607513, -0.028418269008398056, 0.3538084924221039, -0.5281614661216736, 0.24136127531528473, -0.11044559627771378, 0.013703135773539543, -0.49373167753219604, -0.4536263048648834, -0.26134803891181946, -0.7832080721855164, -0.04877819865942001, 0.0768992155790329, 0.9674191474914551, 0.03327566012740135, -0.02572709135711193, -0.4199179708957672, 2.4975979328155518, -0.03338007256388664, 0.03597087785601616, 0.17151828110218048, -0.007108492776751518, 1.4840071201324463, -2.196978807449341, -0.037618301808834076, -0.14433711767196655, -2.084923505783081, -1.143209457397461, -2.0530827045440674, -0.1275922656059265, -0.6312373876571655, -1.3740640878677368, -0.6541182398796082, 0.36439865827560425, 0.04993603006005287, 0.11738619953393936, 0.09082706272602081, -1.5330891609191895, 0.9279585480690002, 0.6417209506034851, -0.26649436354637146, -0.12722980976104736, 0.05164331942796707, 3.226722240447998, 0.0010377452708780766, -0.6899219155311584, 0.298997700214386, -0.20314975082874298, 0.9756263494491577, -0.4094506800174713, -0.5710305571556091, -0.6885353326797485, 1.7345741987228394, 0.07446358352899551, -0.3964620530605316, -0.07722547650337219, -0.9832121729850769, 0.7014575004577637, 0.5636492371559143, 0.3117910325527191, 0.6423481702804565, -0.0747196227312088, 0.6539803743362427, 0.35024771094322205, 0.18178494274616241, -0.10256090015172958, -0.059576839208602905, -1.0991827249526978, 1.5006558895111084, 1.1902779340744019, 0.629978358745575, 0.13748596608638763, 0.309352308511734, -0.07504039257764816, -2.12062931060791, 0.8275496959686279, 1.0256109237670898, -0.00907845888286829, -0.25384214520454407, 0.6877270340919495, 0.0221012681722641, 0.7187071442604065, -0.9456191658973694, -0.1358911097049713, -0.05051166191697121, -0.9929780960083008, -0.06290463358163834, 0.05427432432770729, 2.0267727375030518, -0.9191713333129883, -0.44201070070266724, 0.7012342810630798, -1.1028823852539062, 1.018614411354065, 0.537534773349762, -1.7129465341567993, -0.9697374701499939, 0.07200279086828232, 0.06592612713575363, 1.604148507118225, 0.005820004269480705, 0.018632758408784866, 0.14059682190418243, 1.0102635622024536, 0.029835626482963562, 0.03632264584302902, -1.619062900543213, 0.29164841771125793, -0.48007848858833313, -0.5874409675598145, 2.5718204975128174, -1.240544319152832, -0.6152187585830688, -0.12210504710674286, -0.47384583950042725, -0.7322170734405518, -0.42632776498794556, 0.33019497990608215, -0.4131951630115509, -0.2674485146999359, 0.2727815508842468, 0.03901517763733864, 0.1973864883184433, 1.0882935523986816, 0.18063990771770477, 0.8409163951873779, 0.27096235752105713, 0.13688570261001587, 0.12166654318571091, 0.22428910434246063, -1.713209629058838, -0.30606701970100403, -0.08463160693645477, -7.496299804188311e-05, 0.041086871176958084, 0.06863808631896973, 1.0619335174560547, 0.3098386526107788, -1.250136137008667, -0.007757422514259815, 0.5106852054595947, 0.2748205363750458, 0.019568324089050293, -0.8400909304618835, 0.5716503858566284, -0.23453010618686676, -0.39837297797203064, 2.4547946453094482, -0.51364666223526, -0.6733154058456421, 0.20058408379554749, -0.4944997727870941, -0.5974360108375549, -0.8070407509803772, -0.388467401266098, 0.10606469213962555, -1.25372314453125, 0.24212853610515594, -1.2987391948699951, -0.09002051502466202, -0.37822040915489197, 0.884342610836029, -0.1309068351984024, -0.017047375440597534, -0.4489764869213104, 0.22577330470085144, 0.7847998142242432, 0.0027140886522829533, -0.00426771305501461, 0.06341153383255005, 0.4104640483856201, -1.1502275466918945, 2.364658832550049, -0.9918143153190613, 0.6906248331069946, -0.12226369231939316, 0.25623732805252075, 0.10112220048904419, -0.8541072010993958, 1.7641524076461792, -0.5791153311729431, 0.9103825688362122, 0.9603392481803894, -1.6167250871658325, -0.01652934029698372, -0.6209672689437866, 0.21792596578598022, -3.463425397872925, -0.11649498343467712, -0.7492129802703857, 0.68455570936203, 0.04109081253409386, -1.0265612602233887, -0.015454224310815334, -0.036290671676397324, 3.4957330226898193, 0.3235504627227783, -0.3825279176235199, 0.16581721603870392, -0.0989886075258255, -0.11797255277633667, -0.08387460559606552, -0.44920414686203003, 1.4405733346939087, -0.21200446784496307, 0.006312844343483448, 0.16755476593971252, 0.3738841414451599, 1.002747893333435, 0.29817503690719604, 1.3755650520324707, -3.9863719940185547, -1.103235125541687, 2.4802355766296387, 2.4471564292907715, 0.02525695413351059, -0.4717736542224884, 0.4186863899230957, 0.2817380726337433, -1.1650607585906982, -1.6059174537658691, -0.2548465132713318, -0.07093992084264755, 0.2632923126220703, 0.18236319720745087, -2.9699201583862305, -0.06402214616537094, -0.8926559090614319, 0.653846800327301, 0.09136508405208588, 0.013937171548604965, -2.1995465755462646, 0.071780726313591, 2.3239552974700928, -0.05459759384393692, 0.14512892067432404, -0.8632978200912476, 0.022238776087760925, -0.02701675333082676, 1.8549526929855347, 0.050148431211709976, 1.3349878787994385, 0.06808630377054214, 0.012100028805434704, -0.5200608968734741, -0.39561545848846436, 0.22691644728183746, 0.11574798822402954, -0.12793554365634918, 2.029707193374634, 0.1234804168343544, -0.7848314642906189, -1.0906449556350708, -0.6561890840530396, -2.7253506183624268, 0.022253593429923058, -1.5107115507125854, -1.194481372833252, -0.052143219858407974, -0.0025947114918380976, 2.1636226177215576, -0.017463574185967445, -0.08948791772127151, 0.5294755697250366, 0.8267401456832886, 0.07647573202848434, 0.2857377231121063, -0.05463619530200958, -0.02361896261572838, 0.5106160640716553, 0.6946824193000793, -2.613495111465454, -0.3484371304512024, 1.2767322063446045, -1.4067890644073486, -0.031442757695913315, 0.01875077560544014, -1.2119309902191162, -0.12221835553646088, 0.4518319368362427, -0.8705172538757324, -0.1631973385810852, 3.295969247817993, -0.23907910287380219, -0.24499168992042542, -2.028578042984009, -0.07528340816497803, 0.1527576744556427, -0.5363423228263855, 0.6521170735359192, 1.3519854545593262, -0.7148045897483826, 0.08012443780899048, -0.1391807645559311, 0.06998886913061142, 1.1892586946487427, 0.223112091422081, -1.5169342756271362, 0.6380757093429565, 0.09188859909772873, -0.07715862989425659, 0.6410360336303711, -0.23654361069202423, 1.5702277421951294, -0.11761254072189331, -1.2887910604476929, 0.29139018058776855, 0.8363692760467529, 0.49988508224487305, 2.502786874771118, 1.5204654932022095, -0.5619044303894043, 1.6280184984207153, 0.0027924692258238792, -0.011099487543106079, -0.08533870428800583, -1.287021279335022, 0.6806763410568237, -0.052539050579071045, 0.2635157108306885, 0.5847800374031067, -0.4725189805030823, -0.06889714300632477, -0.009231575764715672, 0.3236153721809387, 0.600944995880127, 1.296622395515442, 2.654060125350952, -2.1914687156677246, 0.162107914686203, -1.6776043176651, -0.2568671405315399, -1.7168292999267578, 1.5362372398376465, -5.462131023406982, -0.8994868993759155, 0.663907527923584, 0.11020714044570923, -0.9064096808433533, 2.1681909561157227, 0.49924370646476746, -0.002240198664367199, 0.5243850350379944, -0.29288774728775024, 0.1629686951637268, 0.8383640050888062, -0.12540994584560394, 1.291430950164795, -1.4073750972747803, 0.035575494170188904, -0.15236571431159973, 1.3992044925689697, -0.5160019397735596, -0.823195219039917, -1.3399817943572998, -0.24649111926555634, 3.294995069503784, 0.06095610931515694, -0.49584826827049255, 0.6248162388801575, 0.25976091623306274, 0.8149064779281616, -1.0712206363677979, 0.010555326007306576, -0.07736241817474365, -0.012947795912623405, -0.004967416636645794, -1.7071514129638672, 0.036902543157339096, -1.0633463859558105, 0.1307152509689331, 1.341134786605835, -2.1740338802337646, 1.398499608039856, -0.19566522538661957, -0.326816588640213, -0.02738865464925766, 0.05269348621368408, -0.5640907883644104, -0.9986439347267151, -0.35872387886047363, -1.825117826461792, 0.03795982152223587, -0.20030377805233002, -0.5048313140869141, 0.5044758915901184, -0.3403204381465912, -0.08089584112167358, -0.1637292206287384, 0.009177706204354763, -0.582708477973938, -3.0424089431762695, 0.07451402395963669, 0.5033115744590759, -0.08152893930673599, 0.11746840924024582, 1.2735892534255981, -1.108667254447937, 1.2546337842941284, 0.13481466472148895, 0.507025957107544, 0.13407261669635773, -0.8569979667663574, -1.5683482885360718, 0.16990837454795837, -0.022311830893158913, -0.7399154901504517, -0.10574375092983246, -0.7341920733451843, 0.6057493686676025, 0.46894732117652893, -0.16238126158714294, 2.052604913711548, 0.7730579376220703, 0.4526495635509491, -0.07788418978452682, -0.25880929827690125, -1.9783709049224854, 0.5077492594718933, -0.03339571878314018, -0.4530378580093384, -0.908694863319397, -1.2053142786026, -0.47698938846588135, 2.871546745300293, -0.504843533039093, -0.4994264245033264, 0.44117575883865356, 0.09607359021902084, -0.4078161120414734, 0.008755202405154705, -2.1319611072540283, 0.009519168175756931, 0.7589221000671387, -0.8356860876083374, -0.5035948157310486, -0.07596113532781601, -2.250333547592163, -0.11242296546697617, -0.32112714648246765, 0.21863143146038055, 0.7383137345314026, -0.1420925259590149, -0.8254821300506592, 0.663421630859375, 1.0119045972824097, -0.3346828520298004, 0.5669989585876465, -1.4679464101791382, 0.1448960155248642, 0.6447740197181702, 1.9808733463287354, 0.004704716615378857, -0.27725526690483093, 0.07037284225225449, 0.0878562182188034, -0.5451799035072327, 2.518423080444336, 0.03094347193837166, 0.1263626664876938, -0.4321082532405853, -1.7655394077301025, 0.715308427810669, -0.19311894476413727, -0.1054234504699707, -0.3093757629394531, -1.3746899366378784, 0.054364871233701706, 0.036892540752887726, 0.16872431337833405, 2.193279504776001, -0.09512116014957428, 0.052013933658599854, 0.5467256307601929, 0.1358444094657898, -0.16698069870471954, -2.274106025695801, -0.6530686616897583, 0.015397358685731888, 1.0838093757629395, 0.22007633745670319, -0.343570739030838, 0.0957154631614685, -0.002092067152261734, -0.857958972454071, 0.16331321001052856, 1.6996963024139404, -0.18054404854774475, -0.3723415732383728, 0.3645358681678772, 0.08613608032464981, 0.6780664324760437, 1.0156053304672241, 0.45845720171928406, -0.8104456067085266, -0.5158719420433044, -0.3320094048976898, -0.7419837713241577, 1.1160457134246826, 0.1903681606054306, 0.10733909159898758, -0.7163547873497009, 0.11418483406305313, 0.3029414713382721, 1.3678390979766846, 0.00019759916176553816, -0.5484347939491272, -0.20876125991344452, 0.7494842410087585, 0.06822708249092102, -1.9623048305511475, 0.13535381853580475, -0.05456722527742386, 0.22978122532367706, 0.07512389123439789, -0.9253919124603271, -0.8650292158126831, 0.31803664565086365, 0.26170840859413147, -0.01032392680644989, 0.04172356426715851, -0.19972218573093414, 0.0629003494977951, -0.06403464078903198, -0.7271959185600281, -0.4196965992450714, 0.528876006603241, -0.19135388731956482, -0.007410647347569466, -0.32218292355537415, -0.045065611600875854, -0.8404662013053894, -0.2353133112192154, -0.6503371596336365, -0.1658165603876114, -0.43395552039146423, -0.6986753344535828, 0.3766016960144043, -0.17047761380672455, 0.053054921329021454, 0.3266424238681793, -1.0952439308166504, 1.3194600343704224, 0.0846823900938034, -4.799920082092285, 1.7610148191452026, 0.2635868787765503, -0.1616114377975464, -0.0643349140882492, 0.0885910764336586, 0.01137920469045639, 0.8055253624916077, 0.634428858757019, 4.296813488006592, -1.421792984008789, -3.173985004425049, 2.0193824768066406, -0.45948147773742676, 0.6722567081451416, -0.4846166968345642, 0.05494260415434837, -1.1143591403961182, 3.9080348014831543, -0.30946773290634155, 0.017353618517518044, 0.03694939240813255, 0.6404378414154053, -0.7803738117218018, 0.2330145239830017, 0.014672340825200081, 0.10401768237352371, -0.3124019205570221, 0.22497086226940155, 2.0267112255096436, 1.30224609375, -0.3292488753795624, -0.8742043375968933, 0.16040392220020294, -1.490024447441101, -0.34937211871147156, 1.025546669960022, 0.3894973695278168, -0.39062514901161194, -0.8634612560272217, 0.01861429587006569, -0.23631708323955536, 0.2189227044582367, 0.49968740344047546, 1.641337275505066, 0.22386114299297333, 0.39221975207328796, 0.49164214730262756, -0.20277956128120422, -1.9040424823760986, -0.18251483142375946, -0.3992636203765869, 0.7705533504486084, 0.596089780330658, 0.1495758444070816, -0.013589262962341309, 0.3119058907032013, 1.2178195714950562, 0.27221378684043884, -0.0033226769883185625, 0.1059698611497879, -0.9931567311286926, -2.871779203414917, -0.038779839873313904, -0.2217177003622055, 0.6734821796417236, -0.20412573218345642, 1.534878134727478, -0.14277753233909607, 1.9647578001022339, -0.1726122796535492, -0.8975102305412292, -0.009239659644663334, -3.5668649673461914, 0.38879114389419556, -0.23200233280658722, -0.046488430351018906, -0.13733720779418945, 0.14508125185966492, 1.7638323307037354, 0.9617887735366821, -0.43389254808425903, -0.46267464756965637, 0.5542635917663574, 0.4277277886867523, 0.447180837392807, 1.050902247428894, 0.0073951794765889645, -0.10574594885110855, -1.135836124420166, 0.48117977380752563, 0.9429327249526978, -0.4316774010658264, -0.2945844829082489, 0.9454473853111267, 0.04448307305574417, -1.0436654090881348, 0.43496936559677124, 0.5310510396957397, 0.823188841342926, -2.3638172149658203, 0.5356706380844116, 0.28050467371940613, 1.156044840812683, 0.6597526669502258, 0.214907705783844, -0.5085591077804565, 0.4278397560119629, 0.27485939860343933, -0.47808578610420227, -0.15938350558280945, 0.0963292196393013, -0.13055971264839172, 1.0454745292663574, -0.49441322684288025, 1.701242208480835, 0.8690471053123474, 0.5560126304626465, -0.007455310318619013, -0.06231295317411423, -1.1485819816589355, 0.7333545684814453, -0.8152210116386414, 0.13075768947601318, 0.1796441376209259, 0.033384550362825394, 0.1633741706609726, 0.5701724290847778, -0.12187564373016357, -0.061335157603025436, 0.032032668590545654, -0.05806651711463928, -0.15836749970912933, -0.038844745606184006, 0.5959177017211914, -0.03668035939335823, -2.1139400005340576, 0.08429459482431412, -0.7989188432693481, 0.3782927989959717, 0.07864227890968323, 0.19561010599136353, -0.24603211879730225, 0.8070780634880066, -0.2728384733200073, -0.7597699761390686, 0.05801341310143471, -0.06606661528348923, -1.2621442079544067, 0.16131798923015594, 0.05413750186562538, 0.06532973796129227, 0.44446539878845215, -0.03779829293489456, 0.01925693266093731, 1.0440291166305542, -0.9315887093544006, -0.009310098364949226, -0.19128181040287018, -0.26938971877098083, -0.17224128544330597, 3.080022096633911, -0.18912622332572937, 1.1292929649353027, 1.8610855340957642, 0.22996722161769867, -0.5589672327041626, -0.24082301557064056, -0.24457961320877075, -0.1940506547689438, 0.71651291847229, 0.965664803981781, -0.08686763793230057, 0.4782472550868988, 0.9310395121574402, -0.8693332672119141, 0.10495983064174652, 0.9828172922134399, -0.8613529205322266, 0.05177311971783638, -0.04549506679177284, 0.46127748489379883, 0.41762596368789673, -0.16928526759147644, -0.5865293145179749, -1.0187538862228394, -1.5375193357467651, 0.9230955839157104, -0.0022674270439893007, -1.544960618019104, -0.02120477706193924, -1.421926498413086, 1.2819650173187256, 0.26287269592285156, 0.08479530364274979, 0.734699010848999, -1.1243181228637695, -0.42911496758461, 0.9500767588615417, 0.6874470710754395, 0.08875640481710434, -0.09726852178573608, 0.48420003056526184, -0.6587612628936768, -1.010688304901123, 2.7674026489257812, 0.1319240927696228, -0.10804934054613113, 0.013127249665558338, -1.544929027557373, 0.015837309882044792, 1.4069498777389526, 0.1694236844778061, -1.7685201168060303, 0.45522019267082214, -0.029409293085336685, 0.3076494336128235, -0.11942575871944427, -0.27419397234916687, -0.08923570811748505, -1.5006210803985596, 0.828140914440155, 0.13015702366828918, 0.2587119936943054, -0.02319757267832756, 0.9420722126960754, -0.49777570366859436, -0.3528372347354889, -0.28724974393844604, 0.001706429524347186, 1.2431377172470093, 0.22809270024299622, 0.04862533509731293, -0.3646430969238281, 1.043113350868225, 0.1340685486793518, -0.6987066268920898, -0.014462120831012726, -0.3026745319366455, -0.31137168407440186, 0.08002021163702011, -0.34615516662597656, 0.041087936609983444, 0.581518828868866, -0.43718647956848145, -0.27929842472076416, -0.4765733778476715, -0.39240625500679016, -0.45920640230178833, -0.3546508848667145, 0.5628767609596252, 0.1867264360189438, 0.2933933436870575, 0.6536427736282349, -3.7207937240600586, 0.7484403848648071, -1.2442748546600342, -1.1037899255752563, -0.0005499570979736745, -1.1411935091018677, -0.029499176889657974, -0.08778521418571472, -0.4287705719470978, -0.4283792972564697, -0.6462547183036804, -1.3160006999969482, 0.005753288045525551, -0.191166490316391, -0.16453997790813446, -0.9454079866409302, 0.43551215529441833, 1.052035927772522, 2.4053518772125244, -0.018407180905342102, 0.20795844495296478, -0.21364425122737885, -0.3244527578353882, 0.21069061756134033, -1.2321959733963013, -1.6156731843948364, -0.07339334487915039, 1.412912368774414, -0.7324655055999756, -1.2145237922668457, -0.16484519839286804, -0.8247752785682678, 3.864107847213745, -1.8306602239608765, 0.10213471204042435, -0.08063157647848129, 0.9494107961654663, -0.9668166041374207, -0.33561477065086365, 1.0044145584106445, 0.5638467073440552, 0.20972664654254913, -1.6328200101852417, 0.45211440324783325, 1.716117024421692, 0.302767813205719, 0.5840272307395935, -0.7035853266716003, -0.4125291109085083, 0.7224538326263428, 0.03037930279970169, -1.282503366470337, 0.07551992684602737, -0.904737114906311, -0.3385513424873352, -0.05990156531333923, 0.030277689918875694, 1.5288480520248413, -0.16800712049007416, 0.14851222932338715, -0.19435633718967438, 0.029924927279353142, -0.6665881276130676, 0.16616681218147278, 0.3185424506664276, 0.22935751080513, -0.4837406575679779, -0.19951869547367096, -1.0975123643875122, -0.8185839056968689, 4.973416805267334, 0.11093771457672119, 0.0007553593604825437, 0.14374655485153198, 6.557300090789795, 0.29768675565719604, 0.2676449716091156, 0.8387938141822815, -0.08017957210540771, 1.7287015914916992, 2.8334991931915283, -0.7173223495483398, -0.6120359301567078, -0.0017831869190558791, 0.8561883568763733, 0.35891294479370117, 1.2897976636886597, 1.1204520463943481, -2.7783172130584717, -0.2985805571079254, 0.05460234358906746, 0.8437584042549133, -0.14254112541675568, 0.14738716185092926, 0.6795457601547241, 1.6770867109298706, 0.8046701550483704, 0.0140884630382061, -1.0967870950698853, -0.04741527512669563, 1.119249701499939, 0.05991542339324951, 0.9583160877227783, 0.3576284646987915, -1.4453569650650024, 0.9653759002685547, 0.0451897457242012, 0.570618212223053, -0.1559104174375534, -0.1271226853132248, -0.008494189940392971, 1.7392324209213257, -0.008055543527007103, 0.6578604578971863, -0.5228884816169739, -0.05235516279935837, -1.118768572807312, -1.4677761793136597, 0.11001425981521606, -0.43614792823791504, 1.6916348934173584, 0.06397788971662521, -1.4674018621444702, -0.019551526755094528, -0.05373053625226021, 1.2233316898345947, 0.22098645567893982, 0.553757905960083, 1.185641884803772, 0.3825969696044922, 0.3624231219291687, -1.7503490447998047, 0.07499140501022339, 0.8328348398208618, -0.03441164270043373, -0.051078520715236664, -2.2960565090179443, 1.1556706428527832, 1.4998785257339478, -0.11697234213352203, 0.14874225854873657, 0.5313700437545776, -2.7189078330993652, -0.35061904788017273, 1.478423833847046, -0.017205631360411644, 0.27255892753601074, 0.6646366119384766, -1.5753012895584106, 0.2208556830883026, 1.1439619064331055, 1.479941964149475, 0.4120146334171295, 0.4533705413341522, 0.03706987202167511, -0.4327259957790375, 0.41123029589653015, 0.15473896265029907, 0.1474621593952179, -0.10701609402894974, -2.978536367416382, -0.5139362215995789, 1.1127655506134033, -1.3657677173614502, 0.7417098879814148, -0.8083367943763733, -0.6820961833000183, 0.05333857983350754, -1.6902554035186768, -0.06302372366189957, 0.7993400692939758, -0.10078296810388565, 0.08268290758132935, -0.48766639828681946, -0.551135241985321, -0.02109174244105816, -0.8034486770629883, -0.24893417954444885, 1.101597547531128, 0.2581118941307068, 3.3620290756225586, -0.9020003080368042, 1.8153226375579834, -1.5926741361618042, -0.42394331097602844, -4.0603203773498535, -1.6589031219482422, 0.013267806731164455, 0.005959727335721254, 0.03399672731757164, 0.00010179241508012637, -0.07094362378120422, 0.154561385512352, -0.6001518964767456, -0.23430387675762177, 0.9146760702133179, 0.08287228643894196, 0.5042487382888794, 0.04978276789188385, -0.060343511402606964, 0.44895148277282715, -0.04210882633924484, 1.6716951131820679, 0.15761633217334747, 0.13097518682479858, 0.5705822110176086, 0.018501851707696915, 0.04282725602388382, -0.2211902141571045, 0.24790625274181366, 1.3845778703689575, -0.02026055008172989, -2.170635461807251, 0.05844076722860336, -0.20389984548091888, -0.19274796545505524, -1.6042896509170532, -3.4148592948913574, 0.1334938108921051, 0.3839573562145233, -0.014662917703390121, 0.0224506426602602, 0.19906717538833618, 0.2927805781364441, 0.02343834936618805, -0.8034000992774963, -0.1061977967619896, 0.5650200247764587, -0.07663272321224213, 0.36651596426963806, -0.04159300774335861, 0.4391714334487915, 0.06331533193588257, -0.8034704923629761, -0.25978049635887146, -0.336101770401001, 0.4518820643424988, 1.6268105506896973, 0.1917949765920639, -0.6303276419639587, 1.9034887552261353, 0.7579036355018616, 0.48652878403663635, -0.33260849118232727, -0.0230556633323431, -1.0891636610031128, 0.030328242108225822, -0.7023855447769165, -1.213523268699646, -0.07885575294494629, -0.21585626900196075, -1.0996357202529907, 0.20092302560806274, 1.0951290130615234, 0.4707253575325012, -0.08045214414596558, -0.009315457195043564, -0.3881385028362274, -3.240704298019409, -0.7603781223297119, 3.5781400203704834, -2.5356006622314453, 0.6816053986549377, 0.8737214803695679, 0.2364211529493332, 2.6773736476898193, 1.2252370119094849, -0.11499129980802536, -1.378831148147583, -0.9899780750274658, -0.1597902476787567, 0.1374201625585556, -1.7673708200454712, 0.017425337806344032, -3.0688164234161377, 0.07340487837791443, -0.7268295288085938, 0.46689674258232117, -0.3319229483604431, -0.14942024648189545, 0.2130659520626068, 1.0650887489318848, -0.8167900443077087, 0.5567611455917358, 1.9621365070343018, -0.6831862330436707, -0.21572010219097137, -0.0657382532954216, -0.2607704699039459, 0.3368660509586334, -0.14455530047416687, -3.1055996417999268, 1.827444076538086, -0.5603581070899963, -0.5408779382705688, -1.7589198350906372, -0.04661251977086067, 0.0017067075241357088, 0.4062216877937317, 0.3422352373600006, 0.6295410990715027, -0.15555673837661743, 0.5173765420913696, -1.3100628852844238, 0.4721699655056, -3.458949089050293, 1.7297743558883667, -1.2298479080200195, -3.219669818878174, -0.09612812846899033, -0.36176127195358276, 0.29842597246170044, -0.009743817150592804, -0.9701071381568909, -0.3438343405723572, 0.6346309185028076, -0.6056191325187683, 0.1337696760892868, -0.571395218372345, -0.6087769865989685, -0.08235897868871689, -0.31990885734558105, 0.08208693563938141, -0.10552902519702911, 0.33987680077552795, -0.5479551553726196, 0.14795416593551636, 1.36118745803833, 0.6845957636833191, -0.07629935443401337, 1.035568356513977, 0.02481243945658207, -0.26052698493003845, 0.01059215608984232, 0.6899620294570923, 0.2769151031970978, -1.3425793647766113, -0.46243634819984436, -0.5999129414558411, -0.5162380337715149, 0.3940838873386383, -0.06633403897285461, -0.6909148693084717, -0.06865861266851425, -3.304288864135742, 0.017575766891241074, 0.15170903503894806, -0.22795170545578003, -1.455862283706665, -0.41691872477531433, 0.21937032043933868, -0.8550501465797424, -0.8316658139228821, -0.08745010942220688, 0.7718746662139893, -0.8859092593193054, 0.9828106164932251, 1.6539716720581055, -0.09414168447256088, 1.184766173362732, -0.8025383353233337, -0.0982707217335701, 0.14914554357528687, 0.03740678355097771, 0.5177627801895142, 0.1997372806072235, -0.139816015958786, 1.6029306650161743, 0.032891131937503815, -0.8536521792411804, 0.2887737452983856, -0.36611098051071167, -0.09427361935377121, -0.9906048774719238, -0.07647702842950821, -0.11853078752756119, 0.09378998726606369, -0.1538868546485901, 0.2942069470882416, -0.30859827995300293, -0.011170812882483006, 0.19295744597911835, -0.3411942720413208, -0.11089121550321579, 0.06469577550888062, -0.6958866119384766, 0.35364943742752075, 0.025042925029993057, 0.45488008856773376, 0.10932185500860214, 2.0211877822875977, 0.17105497419834137, 0.1697794646024704, 0.2281799018383026, 0.07721268385648727, 0.6691278219223022, 0.14921067655086517, -0.030773621052503586, 0.060931555926799774, 1.465790867805481, 0.8127334713935852, 1.7328155040740967, -3.5001471042633057, -1.7691680192947388, -0.10837440937757492, 0.4678487181663513, 0.06847525388002396, -0.35629644989967346, -0.057718802243471146, -0.3892987370491028, 0.3209877908229828, -1.1464406251907349, -0.02213817648589611, -0.1211935505270958, 0.1941078007221222, 0.10444032400846481, -0.16213549673557281, -0.03866580128669739, 0.20283976197242737, 0.44810956716537476, 0.048539899289608, -0.4386264681816101, -2.445434808731079, 3.159144639968872, -0.22068120539188385, 0.030503029003739357, 0.5799151062965393, 0.9280717372894287, 2.842057704925537, -0.5448289513587952, 0.0061226896941661835, -2.1678707599639893, -1.4581509828567505, -0.21826905012130737, -0.013728396967053413, 1.279188871383667, 1.0895534753799438, 2.235321044921875, -1.2023186683654785, 0.4851670265197754, -0.045070432126522064, -0.19244107604026794, 0.004073354881256819, 0.9718934893608093, 0.18037685751914978, -1.68317449092865, -0.25117552280426025, -0.1366414874792099, -1.2601813077926636, -1.2748918533325195, -0.2689250111579895, 0.575780987739563, -1.0560053586959839, -1.1508346796035767, -0.26004326343536377, -0.8691734075546265, 0.026006033644080162, 0.19225585460662842, -0.42231321334838867, -0.2353621870279312, 0.14233824610710144, 3.5105185508728027, 0.10672558844089508, -0.4619143605232239, -0.06595954298973083, -2.128485679626465, -0.08175062388181686, 1.4572702646255493, 0.28763532638549805, -0.17694193124771118, 0.05662032589316368, -0.6560271382331848, -1.06559419631958, -0.009713022038340569, -0.32645437121391296, 0.12685136497020721, 0.6207588315010071, 0.31019166111946106, -0.2688421308994293, -1.1255582571029663, -0.9636396169662476, -1.9478604793548584, -0.3989412188529968, 0.11653012782335281, 2.2819387912750244, -0.00031991919968277216, 0.7618260383605957, 0.05042380467057228, -1.1847654581069946, -0.9962205290794373, 0.39596930146217346, -0.12891899049282074, -1.5770868062973022, -2.7840230464935303, -0.9740059971809387, 1.024642825126648, -0.052740130573511124, -0.39248907566070557, 0.08216163516044617, -2.315822124481201, -0.049160756170749664, 0.3661319315433502, 0.542029857635498, 0.02760203555226326, 1.9526995420455933, -0.3925069272518158, 4.785266876220703, 0.02067556418478489, 0.44357752799987793, -3.84879469871521, -0.3135625422000885, -0.09135822206735611, 0.024069400504231453, -0.19566470384597778, -0.8856208324432373, 0.9347098469734192, 0.048937682062387466, -0.73931485414505, -0.40323132276535034, 0.19272422790527344, -0.03186843916773796, 0.024188367649912834, 1.703337550163269, 0.12336207181215286, 0.22161749005317688, 0.0299571193754673, -0.620856761932373, 0.19984951615333557, 0.1514977514743805, 0.5455734729766846, 1.483345627784729, -0.4461233913898468, 0.11714388430118561, -0.1602153182029724, -0.9249681234359741, -0.42367199063301086, 0.03737477585673332, -0.03372504189610481, -0.9554430842399597, 0.027659261599183083, -0.7266349792480469, -0.061887383460998535, -0.1476317197084427, 0.14559699594974518, -0.0903128832578659, -0.02690327912569046, -1.2994393110275269, -0.942550778388977, 0.03559837117791176, 0.08731282502412796, 0.35039061307907104, 0.4295644164085388, 1.6791037321090698, 0.36214107275009155, 0.26024967432022095, -0.5120443105697632, 1.3359118700027466, -0.015510219149291515, -0.7030094861984253, 0.6221606731414795, -0.10010827332735062, -0.014363636262714863, -0.14625431597232819, -2.173140287399292, 2.2703680992126465, -0.0015737593639642, 2.460188150405884, -0.6033676266670227, 0.32222780585289, 0.104216068983078, -0.6576471924781799, 0.1246010810136795, -1.2939714193344116, -0.02589596062898636, 1.3303292989730835, 0.3005598187446594, 0.09422171115875244, 0.23703007400035858, 0.8539493680000305, -0.4152907133102417, 1.2827026844024658, 0.1746167242527008, 0.07330368459224701, -0.03403598815202713, 0.08445373177528381, 0.9454390406608582, 0.799251914024353, 0.39890116453170776, 0.09104064106941223, -0.03938291594386101, 0.10619326680898666, 2.5911052227020264, 0.7474033236503601, 1.7463306188583374, -2.8557288646698, 3.506577491760254, -0.2294837385416031, -0.41460010409355164, -0.11333443969488144, 1.752989649772644, 1.055545449256897, 0.3970182240009308, 1.2593177556991577, 0.5083996057510376, -0.1844959557056427, -0.5592288970947266, 2.852799415588379, -1.0181381702423096, 0.6844817399978638, 0.1345043182373047, -0.1756054162979126, -4.421712875366211, -0.04886776953935623, 0.15343698859214783, 0.554835855960846, -0.20232351124286652, -0.592894434928894, 0.03349986672401428, -0.010420819744467735, -0.4377647340297699, -0.573945939540863, -0.492345929145813, -0.12934483587741852, 1.002447247505188, -1.1157913208007812, -0.20149913430213928, 0.045250650495290756, 0.009880575351417065, -2.044167995452881, -0.49260103702545166, 0.7178272604942322, 0.3250512182712555, -0.25841987133026123, 0.7492215037345886, 1.9615137577056885, -0.5869784951210022, 1.7210943698883057, 0.17893777787685394, -0.11321146041154861, -0.4398030638694763, -2.1199090480804443, 0.006441236007958651, -0.8499786853790283, 0.6448601484298706, 0.02045268565416336, 1.8142571449279785, 0.44071605801582336, -0.47877827286720276, -0.767195999622345, 0.36375463008880615, -0.12627442181110382, -0.09680969268083572, -2.0040123462677, -0.006032859440892935, 1.8788613080978394, -0.5100109577178955, -0.37221285700798035, 0.41114458441734314, -0.2589907646179199, 0.21396681666374207, 0.33639854192733765, 1.475364327430725, -0.17498059570789337, 1.8028451204299927, 1.3051482439041138, 0.02055078186094761, 0.07442092150449753, 0.1355544477701187, 0.6351547837257385, 0.25453701615333557, 0.1300780475139618, -1.107467770576477, 1.150985836982727, -1.5381108522415161, -0.688506007194519, 0.9249609708786011, 0.027068911120295525, 0.2502853274345398, -0.01665753312408924, 0.0005010539316572249, 0.12463618814945221, -2.6440470218658447, 0.026577342301607132, -4.367880344390869, 1.63454008102417, -0.07754579931497574, -0.7003149390220642, -0.026110831648111343, -0.37036755681037903, -1.8897603750228882, -0.1527712345123291, -0.7528902292251587, -0.6489356160163879, 0.2997705340385437, -0.5207863450050354, -0.3980183005332947, 0.4392377436161041, 0.38497477769851685, 0.08194983750581741, 0.21184559166431427, 0.0955447107553482, 0.6425336003303528, 0.2899296283721924, 2.66086745262146, 1.7546215057373047, -0.0033105455804616213, 0.25646138191223145, -0.01626838743686676, 0.9593137502670288, -0.48917821049690247, -1.419399380683899, -0.10863583534955978, -0.013583787716925144, -0.007078512106090784, -0.264101505279541, -1.4965366125106812, -0.4607706069946289, -1.1940290927886963, 2.6554629802703857, 0.5658264756202698, 0.4010729193687439, 0.3412656784057617, -0.11776452511548996, 2.995704412460327, -0.6896002292633057, -0.4602549076080322, -0.10560517758131027, 0.5791162252426147, -1.700875997543335, -0.06450311839580536, -0.10904952138662338, -0.4052194058895111, -0.6073422431945801, 0.26758503913879395, -0.009455228224396706, -0.055337049067020416, 2.6968436241149902, -0.7332072854042053, 1.1537595987319946, 0.9204146265983582, 0.7536795139312744, -0.11817997694015503, -0.5790596008300781, 0.056091830134391785, 0.13235536217689514, 1.4666365385055542, 0.054586950689554214, 0.042700354009866714, 0.21077357232570648, -0.11461762338876724, -0.22851254045963287, -0.20724613964557648, -0.6039246916770935, 0.017885908484458923, 0.279568076133728, 0.21757976710796356, 0.025384286418557167, 0.8194160461425781, -0.4429354965686798, -0.7131776809692383, -0.2203485667705536, 0.17127388715744019, 0.1520358771085739, 0.037684690207242966, -0.8566573858261108, -0.03809484839439392, 0.5263205766677856, -0.130561962723732, -1.14704430103302, 0.4696015417575836, 0.5488176941871643, -0.5777489542961121, 3.876760482788086, -1.790626049041748, -0.3000418245792389, 0.42385974526405334, -0.3093474209308624, -0.0053312210366129875, 0.46922340989112854, 0.03187010809779167, 0.3853534162044525, 0.1345902979373932, -0.8844938278198242, 0.3219372034072876, -0.9089733958244324, -0.36309128999710083, -0.016887519508600235, -0.08534882962703705, 1.7497931718826294, 0.3874814510345459, -0.03481390327215195, -0.18326076865196228, 1.272194743156433, -0.23854777216911316, 1.095101237297058, -0.10811810940504074, -0.12454840540885925, 0.07737338542938232, -0.2099130004644394, 0.0039496272802352905, -1.9064503908157349, 0.42361050844192505, 0.11971818655729294, 2.4492197036743164, 0.48130112886428833, 0.637891948223114, -0.3275753855705261, 0.7450357675552368, 0.3825261890888214, -1.5726070404052734, 0.0867847129702568, -0.26921546459198, -0.07420594245195389, -0.3565470278263092, 0.12691012024879456, -0.22066780924797058, -0.6233050227165222, -1.0418859720230103, 1.5174051523208618, -1.860192894935608, -2.3298146724700928, -0.3624924123287201, -0.44724297523498535, -0.3366366922855377, 1.8115953207015991, 0.780149519443512, 0.961052417755127, 0.11815787851810455, 0.5534127354621887, -0.03303449600934982, -0.5078310966491699, -0.4621453583240509, -0.3400942087173462, -0.4400027096271515, -0.5788531303405762, -0.36967065930366516, 0.3197600543498993, -1.291831374168396, -0.1343431919813156, -0.38732627034187317, -0.3773060142993927, 0.30649420619010925, 0.7049350142478943, -2.6137402057647705, 0.20673201978206635, -0.955675482749939, 0.2511652410030365, -1.6762632131576538, -2.152033567428589, -0.11089865118265152, -0.9426306486129761, 0.5965331792831421, 0.06560063362121582, 0.13185766339302063, -0.6205801367759705, 2.4136765003204346, -0.13193672895431519, 0.14806880056858063, -2.576150894165039, 1.7145650386810303, -0.06357485800981522, 0.020515531301498413, 0.10894811898469925, -0.006078149192035198, -0.3079115152359009, 0.0019618344958871603, -0.37503302097320557, 1.9534480571746826, -0.25841864943504333, 1.613011121749878, 0.11338914930820465, 0.02148623950779438, 0.750804603099823, 0.1258687674999237, -0.3392598628997803, 0.04307283088564873, -1.8969250917434692, 0.43268710374832153, -0.7570178508758545, 0.12961286306381226, 0.037319354712963104, -0.5551271438598633, 1.0534144639968872, -0.11353129893541336, 0.03052748739719391, 0.041532669216394424, 0.0007481019129045308, 0.1038280725479126, 1.4338406324386597, -0.08117736130952835, -0.010138445533812046, -0.5406566858291626, -0.3016601800918579, -0.5331675410270691, -0.4746149480342865, 0.3395519256591797, -0.02086358144879341, 0.34907352924346924, 0.06760571151971817, 0.18240666389465332, -0.7552633285522461, -0.03801965340971947, 0.0002445977879688144, -0.4719649851322174, -0.26117947697639465, 0.41094499826431274, 0.978306770324707, 3.3382606506347656, -0.9750451445579529, 0.3873647153377533, 0.1313469558954239, 2.6644387245178223, -2.3325939178466797, 0.10748717933893204, -0.013786396943032742, -0.1167369931936264, 0.9103826880455017, -0.32397955656051636, -0.1857348084449768, 0.00041461168439127505, 0.11473513394594193, -1.6104191541671753, 0.16631636023521423, 0.627260148525238, -0.03912563621997833, -0.1051194965839386, 0.3015911281108856, -0.13898015022277832, -0.12154673784971237, -2.7377424240112305, -0.0026124678552150726, 1.3847934007644653, 0.0035777995362877846, 1.1289302110671997, -0.05836134031414986, -0.6353830099105835, 1.8042229413986206, -0.7524446249008179, -0.007538835518062115, 0.7910202741622925, 0.3360176980495453, -0.09071069210767746, 0.2062833160161972, -1.579809308052063, -0.7135555744171143, 2.222926139831543, -0.5501444339752197, -3.663444757461548, 1.5938085317611694, -2.9750213623046875, 1.1534441709518433, -0.103410504758358, -0.19586727023124695, 1.6217128038406372, 0.26461517810821533, -1.5204094648361206, -0.3173381984233856, -2.207442283630371, 0.559019923210144, -1.733532190322876, -1.7623023986816406, -0.3538892865180969, -0.02495497465133667, 0.0740671157836914, -1.451778531074524, -0.14811384677886963, 1.5585885047912598, 0.18492282927036285, -0.010848836973309517, -0.3133600056171417, -0.21556499600410461, -1.4933642148971558, -0.19841456413269043, -0.96250981092453, 0.37186434864997864, -0.8219538927078247, -0.062198102474212646, 0.06349283456802368, 0.2519892454147339, 0.6354933977127075, -0.3645937144756317, -0.2821243703365326, 1.2153527736663818, 0.003713178215548396, 0.4603535234928131, 0.6958222389221191, -0.03569883108139038, -0.005359600763767958, -0.39292722940444946, 1.1326044797897339, -0.07984849065542221, 0.2821660339832306, 0.20917876064777374, -0.7131640911102295, 0.016102075576782227, 0.21957716345787048, -0.8578276634216309, 0.061578214168548584, -3.0639119148254395, 0.9341495037078857, 1.7187988758087158, -0.15689246356487274, 0.003022489370778203, -0.42591023445129395, 0.5693806409835815, -0.144867941737175, 0.8122279644012451, 0.040707364678382874, 0.9678281545639038, 0.6337593793869019, -0.22414091229438782, 3.053288221359253, 1.8537293672561646, 0.7953459620475769, 1.1820696592330933, -0.692791759967804, 1.2014400959014893, -0.07861226797103882, -0.0351322665810585, 2.0452492237091064, 0.03380538895726204, -0.17696082592010498, 0.7493894696235657, -0.009473398327827454, 0.1036066934466362, 0.08185120671987534, -0.22314243018627167, -0.15352049469947815, -0.4066557288169861, -0.3022802770137787, -0.2743239402770996, -0.134684219956398, 1.0960829257965088, -1.409930944442749, -0.220988929271698, -0.26365941762924194, -0.40713366866111755, -0.16283132135868073, 0.5482223629951477, -0.0037006139755249023, -1.433890461921692, -0.3985673487186432, -1.0850591659545898, -0.4245326519012451, -0.117678701877594, -0.05930117890238762, 0.10254837572574615, -0.1484924554824829, 0.06798769533634186, 0.2680599093437195, 0.37836703658103943, 0.28364333510398865, 0.45107898116111755, -0.48619991540908813, 1.8958511352539062, 0.4611313045024872, -0.864412248134613, 0.034125279635190964, -0.31606781482696533, -1.0434026718139648, 1.3123342990875244, -1.1783852577209473, -0.04318184405565262, 0.5897464752197266, 0.09392344951629639, -0.4216897189617157, -1.3158637285232544, 0.8810040354728699, -0.5279573798179626, 0.1625857800245285, 2.410276412963867, -1.0488505363464355, 1.1413766145706177, -1.846176028251648, -0.24379709362983704, -0.7342234253883362, -0.5625174045562744, -0.3548055589199066, -1.5614629983901978, -0.4794107675552368, -0.0007974269683472812, -1.3826485872268677, -0.5177755951881409, -0.12231138348579407, 0.02294710837304592, 0.020429139956831932, 0.6896966099739075, -1.9886378049850464, 1.5682185888290405, 0.17427051067352295, -0.11082153767347336, 0.7851671576499939, 0.0018453303491696715, 0.12098696082830429, 0.2872268259525299, -0.0010702410945668817, 0.7480372190475464, -0.3066180944442749, 0.0946166142821312, 0.3480933606624603, 0.15552695095539093, 0.6863729953765869, -0.32428258657455444, 0.28535956144332886, 0.07891710102558136, -0.9625013470649719, 1.2057920694351196, -0.15321676433086395, 5.20387077331543, -0.18869438767433167, -0.7855958342552185, 0.1761702597141266, -0.4279050827026367, -0.25777706503868103, -0.22494734823703766, 1.7590886354446411, 0.026921821758151054, -0.8009341359138489, -0.02266690693795681, 0.2477717250585556, 0.4954483211040497, 0.33876368403434753, -0.29728755354881287, 4.015729904174805, -0.5163301229476929, -0.001936248387210071, 0.27955499291419983, -0.17649149894714355, 3.5749385356903076, -1.8581875562667847, 0.18385060131549835, -1.7068915367126465, -0.3700806498527527, 0.8515336513519287, 0.10071048140525818, 0.5820305943489075, 0.7647197842597961, -1.3819783926010132, 0.5703886151313782, 0.21065963804721832, -0.2696134150028229, -0.010761496610939503, 0.08695702999830246, 0.7068514227867126, -0.007842378690838814, 0.7375960350036621, 1.5593727827072144, -0.6273608207702637, 0.06705630570650101, -0.03362635523080826, 1.546795129776001, -0.1308220773935318, 0.010697628371417522, -0.09896396100521088, -0.3240862786769867, -0.7061725854873657, 1.1271734237670898, 3.7520697116851807, 1.3057621717453003, -0.2602072060108185, 0.13232730329036713, -0.14481893181800842, 0.08384136110544205, -0.08325646072626114, -4.554786682128906, 1.3236256837844849, -0.6646761298179626, -0.4025786519050598, 0.8589423298835754, -0.025569118559360504, 0.5531354546546936, 0.28238025307655334, -0.10073443502187729, 0.17276163399219513, 0.1648368090391159, -0.2454417198896408, -0.00899101048707962, -0.7505674362182617, 0.7680279016494751, -1.5302445888519287, 0.8230118155479431, -0.01764642260968685, 0.7178134322166443, -1.0635584592819214, 2.8526244163513184, -0.31033003330230713, -0.04883742332458496, -0.10474193096160889, -1.0446730852127075, -0.4744032025337219, -0.23844656348228455, -0.48978978395462036, 0.9044741988182068, 0.016815045848488808, -0.7269904613494873, -0.9859188795089722, 0.2956489622592926, -1.7957919836044312, 0.012886952608823776, 0.09906021505594254, -0.7127521634101868, -0.28725823760032654, 1.586065649986267, -0.11033477634191513, 0.1000022366642952, 0.45048680901527405, -2.253291130065918, 0.30249273777008057, -1.116014838218689, -0.1732400804758072, -1.449246883392334, 0.08760812878608704, 0.23904810845851898, 0.5823651552200317, -0.2770087420940399, -2.0746207237243652, -0.5245329141616821, 0.22174464166164398, -0.2937581539154053, 0.3330287039279938, -0.22403886914253235, -0.014722325839102268, -0.014236023649573326, -0.19094893336296082, -0.40192002058029175, 1.287024736404419, 0.026827391237020493, -0.30670061707496643, -0.9964377284049988, -0.003946254029870033, -0.4102395176887512, 0.026054635643959045, 0.23732703924179077, -2.5583269596099854, 0.08997894078493118, -0.14988860487937927, -0.8837740421295166, 0.04554872587323189, -3.7423346042633057, -0.2880709171295166, 0.21222837269306183, 0.5986796617507935, -1.2887518405914307, 0.9747788310050964, 0.0627361536026001, 0.00015108859224710613, 1.538544774055481, 0.7717083096504211, -0.012135910801589489, 0.5669158697128296, 1.930431604385376, -0.4908424913883209, -1.4592691659927368, 1.0027714967727661, 0.24956627190113068, -0.3077826201915741, -0.49309155344963074, 0.45227131247520447, 0.07692071050405502, -0.204071044921875, -0.5585643649101257, -0.3435627818107605, -0.653682291507721, -0.5897736549377441, -1.0085660219192505, 0.41325363516807556, -4.424444198608398, -0.14573846757411957, -0.14935658872127533, 0.1668035238981247, -0.438941091299057, -0.5286015272140503, 0.7988765239715576, -0.572590708732605, 0.012721290811896324, 0.2773030400276184, 0.021885497495532036, 0.2658396363258362, 0.13490988314151764, 0.01725262589752674, 0.5196879506111145, -0.2735459506511688, 0.558026134967804, -0.7639274001121521, 0.21436256170272827, -0.04959665983915329, 2.503875970840454, -1.0664883852005005, -0.011563939973711967, 0.06131083518266678, 0.18959730863571167, 0.47616657614707947, -1.1666374206542969, 0.964313268661499, -1.9100959300994873, -1.3294850587844849, 0.02675587311387062, 0.240647092461586, 1.5854078531265259, 0.22031807899475098, -0.0869145542383194, 0.031807791441679, 0.02659395895898342, -0.6527459025382996, 0.005677871871739626, -1.7169512510299683, 0.7275786995887756, 0.0029746592044830322, 0.8158146142959595, -0.6991292834281921, -0.7507339715957642, 1.4045675992965698, -0.5476894974708557, 0.11580374836921692, 0.05907757580280304, 0.3561425805091858, 0.4549488425254822, -0.07870098948478699, 1.8024404048919678, -0.03644362837076187, 0.7355915904045105, 0.41064372658729553, 0.6408997774124146, 0.3734325170516968, 0.923028290271759, -0.3013235926628113, -0.16831587255001068, 0.42105716466903687, -0.6574288010597229, 0.022169845178723335, -0.18096067011356354, 0.09029451012611389, 1.4030638933181763, 0.02337222546339035, 0.08065985143184662, -1.0494451522827148, 0.05634741485118866, -2.314274311065674, 1.1880594491958618, -0.49480780959129333, 0.003617971669882536, -0.10109115391969681, 0.19440284371376038, -0.0013156709028407931, -0.06686479598283768, -0.31431230902671814, -0.14397571980953217, -1.8597263097763062, -0.36876341700553894, 0.41091346740722656, -0.0023107128217816353, 0.24843144416809082, -0.9378087520599365, -1.141663670539856, -3.6133103370666504, -0.7735724449157715, 0.707455575466156, -0.8052536845207214, -0.06678610295057297, -0.07861930876970291, -0.878318727016449, 0.12082900106906891, 0.035121768712997437, -0.3041335642337799, 0.031941816210746765, -0.16306303441524506, 0.22526226937770844, 0.04025725647807121, -0.16944773495197296, 0.20931485295295715, 0.8468340635299683, -0.1675765961408615, -0.007822391577064991, 2.6523189544677734, -0.03312213346362114, 0.9495114684104919, 0.37068742513656616, 0.36915332078933716, 1.2468093633651733, -0.3064599633216858, -1.3689684867858887, 0.6228513121604919, -0.17850413918495178, -0.2392978072166443, 2.3873729705810547, 0.09759726375341415, -0.626386284828186, 0.40041643381118774, 0.0027861010748893023, 0.06266972422599792, 0.022551149129867554, 0.7409343123435974, 0.09531994163990021, 0.0071551441214978695, -0.20475609600543976, 0.8067234754562378, 0.1940903216600418, 0.5010364651679993, 1.7267377376556396, -0.4881548583507538, 0.08275792747735977, -0.3951564431190491, -0.8959311842918396, 0.20704655349254608, 0.010243915021419525, -0.29775741696357727, -0.10982594639062881, 0.037705857306718826, 0.3103049695491791, -0.1716596633195877, -0.18804682791233063, -1.611711859703064, 1.3256763219833374, 0.037825264036655426, -0.9227985143661499, -0.3963744044303894, 0.6633719801902771, 0.14405955374240875, 0.8035053610801697, 0.3402269184589386, -0.062439944595098495, 0.1533338725566864, -0.5886496305465698, -1.639874815940857, 1.8186169862747192, -0.5512939691543579, -0.00500858947634697, 0.0074538737535476685, 0.6940244436264038, 0.8358076214790344, 0.35302552580833435, 0.16020259261131287, 0.933810293674469, 1.1112185716629028, 0.1314573884010315, 0.05068592354655266, 0.2770753502845764, -1.9445950984954834, 0.04264647513628006, -0.5217157006263733, 1.2504231929779053, 1.4973057508468628, -0.07485160231590271, -0.3561079800128937, -0.23176966607570648, 0.014153774827718735, -0.1100042536854744, -0.45821303129196167, -0.09558791667222977, -0.7982929944992065, 1.89244544506073, -1.1669158935546875, 1.7383233308792114, -0.5875666737556458, -0.23868438601493835, 0.5078339576721191, 4.059043884277344, -0.009906942024827003, -1.1648503541946411, -0.062324922531843185, 0.1071012020111084, 0.19035640358924866, 1.2664307355880737, -1.0704227685928345, -1.583935260772705, -0.36003944277763367, 0.010867436416447163, -0.27292725443840027, -1.7010754346847534, -0.07807382941246033, -0.07458244264125824, 0.22984710335731506, -0.009323257021605968, -1.8238945007324219, -0.21863675117492676, 0.2589021623134613, -0.0764857605099678, -0.39347749948501587, 1.0142815113067627, 0.15779277682304382, -0.330675333738327, -0.6023451685905457, -0.20564770698547363, -1.2943559885025024, 0.6184314489364624, -0.005848064552992582, 0.11054922640323639, 0.8074568510055542, 0.07276807725429535, 0.04876338690519333, 0.17659226059913635, 0.3100742697715759, -1.3933286666870117, 0.21573348343372345, -0.04589175805449486, 0.16276302933692932, -0.267110139131546, 0.07105352729558945, -0.31439709663391113, 0.37918663024902344, 0.1109718605875969, 2.6887357234954834, -0.1981947422027588, 0.9823400378227234, 0.02881644293665886, -0.27025124430656433, -1.551352620124817, 0.1825915426015854, 0.030025865882635117, 0.01404882874339819, -0.3172925114631653, 0.11442511528730392, -0.37415701150894165, -0.6198654770851135, -0.2500673532485962, 3.190095901489258, 0.027895817533135414, 0.5583676099777222, 1.3049778938293457, 0.11481143534183502, 2.4250011444091797, -0.054282162338495255, 0.058924634009599686, 0.06906440854072571, 0.5306590795516968, -0.1803283840417862, -0.9896774291992188, -0.5532432794570923, -0.4418714940547943, -0.06340474635362625, -0.4283750653266907, 0.1489352136850357, -0.20412060618400574, -2.5426156520843506, -0.34910985827445984, -0.31802213191986084, 1.1173399686813354, -0.8522794842720032, -0.5742849111557007, 0.5883000493049622, 0.6160094738006592, -0.29360976815223694, 0.001449759118258953, 0.6350365877151489, 1.5116475820541382, -0.1560233235359192, -0.27395838499069214, -0.9440286159515381, -0.6625977754592896, -0.6677944660186768, -0.4501118063926697, -0.003581839147955179, -0.5991220474243164, 0.03115067444741726, -0.5854012370109558, -0.7885887622833252, 0.0689467266201973, -0.8783361911773682, 0.16898024082183838, 0.23137931525707245, -0.5094798803329468, 0.04040074720978737, 0.48334628343582153, 0.20141811668872833, 0.0700593888759613, -2.123385190963745, 0.09349105507135391, 0.5834865570068359, 1.1385492086410522, -0.2539982795715332, 0.06711415201425552, 0.5101168751716614, -0.2892613708972931, -0.9764513969421387, -0.31574392318725586, -0.13946282863616943, -0.6900527477264404, -0.31897029280662537, 0.20688413083553314, 0.8451027870178223, 0.7472416758537292, -0.003195846686139703, 0.11581167578697205, 0.8372259140014648, -0.31538230180740356, -0.30702516436576843, 3.8984951972961426, 0.007472224999219179, -0.24877776205539703, 0.194216787815094, 0.08003044873476028, -1.7941713333129883, -0.3198775351047516, 0.5519503355026245, 0.15528009831905365, 0.2142758071422577, 0.16798222064971924, 0.9282127618789673, -0.4608231484889984, -0.280347615480423, -0.5873048305511475, 0.1454094499349594, -0.18716156482696533, 0.9800390601158142, 0.5642082691192627, -0.2470146268606186, 0.022496163845062256, 0.5103521943092346, -0.7247902750968933, 0.06057824194431305, 0.3278299570083618, 1.9511306285858154, 0.14370229840278625, -1.5256990194320679, -1.733299970626831, -0.11081551015377045, -0.13066056370735168, -0.4648541510105133, -1.3549678325653076, -0.6087570786476135, -0.3703356683254242, 0.0018497768323868513, 0.3765829801559448, -0.8768281936645508, 0.15276482701301575, -1.2471905946731567, 0.019913723692297935, 0.42729756236076355, 1.8107209205627441, 0.49726712703704834, 0.2264818698167801, 1.0359530448913574, -0.0939936488866806, -1.5228244066238403, -0.012285901233553886, -0.15380026400089264, 0.767314612865448, -1.082927942276001, -1.3706226348876953, 0.9098473191261292, 0.5163358449935913, 0.5690416693687439, -0.3970438539981842, 0.01939256861805916, 1.8801342248916626, 1.781899333000183, -0.7689090967178345, -0.07742619514465332, -1.1633405685424805, -0.6835504174232483, 0.555934488773346, 0.8487978577613831, -2.6677682399749756, 0.8852763772010803, -0.05483931303024292, 0.09568390250205994, 0.48629820346832275, 0.5779067277908325, -0.06847240030765533, 0.8191762566566467, -2.906564474105835, 0.005137838423252106, 0.0023801058996468782, 0.687860369682312, 0.5085299611091614, 0.22758087515830994, 0.23037728667259216, -0.942081868648529, 0.03711631894111633, -0.7432762980461121, 0.3849090039730072, 0.9875906705856323, 0.06761181354522705, -0.006155133713036776, -1.1955736875534058, 0.11645153909921646, -0.8466965556144714, 0.9132259488105774, -0.14978991448879242, 0.13407640159130096, -0.13927380740642548, -0.8890604972839355, 0.013454251922667027, 0.3390258550643921, 0.027121450752019882, -0.12366712838411331, -0.16169637441635132, -0.24018093943595886, 0.7983059883117676, -0.18384507298469543, -2.9435815811157227, -0.18601039052009583, 0.09399934858083725, 0.4423483908176422, -2.2177233695983887, -0.01317485049366951, -0.35041677951812744, -0.20650434494018555, -0.07296330481767654, 1.1606165170669556, 0.060094207525253296, 1.2063413858413696, 2.0078368186950684, -0.7689841389656067, 1.808546543121338, -0.06440343707799911, -0.09795079380273819, 0.17694038152694702, 0.6401668787002563, 0.07934846729040146, -0.9792969226837158, -0.031890224665403366, 0.16275902092456818, -1.6144362688064575, -0.03582180663943291, -0.5996785163879395, -0.10876379907131195, -1.092142939567566, 0.6075311899185181, -0.546663761138916, 0.13029415905475616, -0.0725884661078453, -0.1710861325263977, -0.6564392447471619, -0.07817219942808151, -0.004056069068610668, -3.3667726516723633, 0.06510263681411743, -1.1589922904968262, 0.42097657918930054, 0.09319791942834854, -6.374070644378662, 0.37903520464897156, -0.2680029571056366, -0.30009371042251587, -0.43902429938316345, 0.377595990896225, -1.2960050106048584, -2.949954032897949, 0.4044579863548279, -1.830444097518921, -0.2096855342388153, 0.3079529404640198, -0.3799598813056946, 1.0029687881469727, -0.10065010190010071, -0.3810860514640808, -0.6060954332351685, 1.1835575103759766, 0.24158617854118347, 0.1442505121231079, 0.18582817912101746, -0.6256546974182129, -0.7115153670310974, 0.01867755688726902, 0.193115696310997, 0.32032880187034607, 0.27736568450927734, -0.07141999900341034, 0.09886905550956726, -2.090512275695801, 0.3176141083240509, -0.05113514885306358, -0.737324059009552, 0.011876044794917107, -0.1510077565908432, 0.5730053186416626, -0.008417217060923576, 0.008923215791583061, 0.5469249486923218, -0.18409989774227142, -0.23841726779937744, 0.033286552876234055, 0.9111417531967163, -0.15849216282367706, 0.862596869468689, 0.062413912266492844, -0.2779459059238434, 0.10807158052921295, 0.02606816776096821, 4.333478927612305, 0.1556769460439682, -0.9519439339637756, 0.39124372601509094, 0.03227175027132034, 0.5186819434165955, 0.7274528741836548, -0.8640440702438354, -0.02788994461297989, 1.5008705854415894, -0.978131115436554, -0.31405648589134216, 0.03548792377114296, -0.06752051413059235, -0.08559728413820267, -0.8311851024627686, 1.2542157173156738, 0.06907065957784653, -0.8009890913963318, 0.5090155005455017, -0.43388256430625916, 0.7922081351280212, 1.8895732164382935, -0.17780917882919312, -0.7952291369438171, 0.6541547179222107, 0.34129801392555237, -0.15881721675395966, -0.5045238137245178, 0.670423686504364, -0.07961048185825348, 0.02053329534828663, 0.03181255981326103, -0.10963498800992966, 1.4823799133300781, -0.22888809442520142, 0.31021222472190857, 0.0049583748914301395, 1.9595134258270264, 3.2527499198913574, 0.0042169708758592606, 0.7766558527946472, 0.03333160653710365, -0.8910633325576782, 5.416358470916748, 1.2567702531814575, -0.15007427334785461, -0.5204153656959534, -0.5755441784858704, 0.2644239664077759, 0.38373297452926636, -0.12477188557386398, -0.25215229392051697, -0.76980060338974, 0.7435132265090942, 1.0840667486190796, 0.05677749961614609, 1.471790075302124, -0.23584014177322388, 0.5681971311569214, 0.46715831756591797, 2.291682004928589, 1.2158344984054565, -0.4109722673892975, -0.14314743876457214, -1.8105500936508179, -0.5166322588920593, -1.7529921531677246, 1.2866082191467285, 0.27746322751045227, 0.9489912390708923, -0.2623310387134552, -0.632519543170929, -0.12956605851650238, -0.1289089024066925, 0.07531069964170456, 0.023697709664702415, -0.9092236161231995, 1.1811940670013428, -0.4129706621170044, 0.785265326499939, 0.9012332558631897, 0.11661064624786377, -0.5267195105552673, 0.7004145979881287, -0.026964861899614334, 1.1030759811401367, -0.21132121980190277, -0.5394434332847595, 1.069624423980713, 0.21489039063453674, 1.8510808944702148, 0.3137248456478119, -0.9187580347061157, 0.624532163143158, -2.5551345348358154, -0.10903693735599518, 0.0013354874681681395, -0.3727533221244812, 1.6945043802261353, -0.32110244035720825, 1.985365629196167, 0.06195354461669922, -0.7655929923057556, -1.3409912586212158, 0.24424517154693604, 0.0008672213298268616, 0.0068900687620043755, 0.18829938769340515, -0.6563847661018372, 1.0707125663757324, -0.8619229793548584, 1.1405354738235474, 0.15725411474704742, -0.37310558557510376, 0.6478272676467896, 0.07842978090047836, 0.20654146373271942, -0.44586411118507385, -0.9904914498329163, -1.4773374795913696, -0.11592204123735428, 0.3244335651397705, 0.10282628983259201, -0.8304206132888794, -0.0844108983874321, -0.5831552147865295, 0.01818404346704483, -0.30385148525238037, 0.2957279086112976, -0.7704908847808838, 0.05352715402841568, -0.008250453509390354, -0.4588320851325989, -1.8471215963363647, 0.10893373936414719, 0.04161566123366356, 0.4617460370063782, -0.5245676636695862, -0.3430509567260742, -0.14831212162971497, 0.09971015155315399, -0.14691373705863953, 1.7848083972930908, 2.3175911903381348, 0.20658482611179352, 0.6198702454566956, -0.34211286902427673, -0.3688909709453583, -0.5535558462142944, -0.8524847626686096, -0.36676496267318726, 0.6627988815307617, 2.382572889328003, -0.22308854758739471, 0.10527827590703964, 0.599204421043396, -0.006664565298706293, -0.7433951497077942, 0.9711775183677673, 0.356649249792099, 0.4595824182033539, -0.16310028731822968, 0.4471314251422882, -0.5103643536567688, -0.017623744904994965, 4.561519145965576, 1.2432254552841187, 0.6426772475242615, 1.253891944885254, -0.18729814887046814, -0.41933712363243103, -0.08406651020050049, -0.03351927176117897, 0.43659189343452454, 0.5870531797409058, 2.2289812564849854, -0.09526757895946503, 0.04780687764286995, 0.3796401619911194, 1.490233302116394, 0.5296929478645325, 0.19507355988025665, -0.018544066697359085, -0.8141648769378662, 0.4907454252243042, 0.5831790566444397, 0.15310558676719666, -0.4815153479576111, 0.13902927935123444, -0.37881654500961304, 0.5930277109146118, 0.4433991312980652, -0.28593021631240845, 0.14044763147830963, -0.6566089987754822, -0.16891665756702423, 0.2943146228790283, -0.06335978209972382, -0.004240776877850294, 0.6436890959739685, 2.0255751609802246, 0.4156979024410248, 0.22112971544265747, 0.03304755315184593, -0.3348556458950043, 0.707390308380127, 0.2850291430950165, -0.4553934633731842, 0.12876468896865845, -0.5538718700408936, 0.8687955141067505, 0.044240567833185196, -0.9629940986633301, 1.4502999782562256, 0.0997830331325531, -0.2618563771247864, 0.45719805359840393, -0.03522896766662598, 0.9525238275527954, -3.449016571044922, 0.36307859420776367], [-0.6516034007072449, -0.39129769802093506, -0.04852123558521271, -0.1243138313293457, -0.448613703250885, -0.14106284081935883, 0.02166174165904522, -0.8032865524291992, 0.9305340647697449, 5.311807155609131, 1.1139283180236816, -0.529458224773407, -0.5279061198234558, 0.1539061963558197, 0.15096288919448853, 0.48045670986175537, -0.37522822618484497, -0.11147855967283249, 0.1643499732017517, 0.31792259216308594, -0.10538134723901749, 0.11622262001037598, -0.02648770622909069, 1.802216649055481, -0.2343921810388565, 0.576324462890625, 0.35643598437309265, 0.03499623388051987, -0.20843486487865448, 0.016487741842865944, -0.9595009088516235, 0.6024674773216248, 0.7715272903442383, 0.0732090175151825, -0.00031182041857391596, -2.90014386177063, 0.28969553112983704, -0.058076873421669006, -0.27517008781433105, 0.9296732544898987, -0.5390925407409668, 0.5932228565216064, -0.08147229999303818, 0.3223811089992523, 0.027378130704164505, -0.5910884737968445, -0.07176890224218369, 0.17345933616161346, -1.617161512374878, -0.3802321255207062, 1.4670251607894897, 0.009251164272427559, 0.13183902204036713, -0.07713355123996735, 0.25778329372406006, 0.5832919478416443, -0.3640395998954773, 0.004495667293667793, -0.1100120022892952, -0.11709881573915482, 0.40301579236984253, 0.016299346461892128, 0.5380658507347107, 1.0181680917739868, -0.6582727432250977, 0.08793672174215317, 0.2458711564540863, 0.43216052651405334, -0.13357260823249817, -1.0415254831314087, 0.08867615461349487, 0.8762548565864563, 0.016561565920710564, -2.539090156555176, -0.4316294193267822, 0.7114152908325195, 0.009370576590299606, -2.883795976638794, -0.04100968316197395, -1.2191630601882935, -1.0749475955963135, 0.3877657651901245, -0.3113381266593933, 0.2810615301132202, 0.28372377157211304, -0.5994142293930054, -0.16789713501930237, -0.21198706328868866, -0.7089057564735413, -1.1328752040863037, 0.3913191854953766, 0.3261653780937195, -0.02652897872030735, 0.11183887720108032, -0.08420291543006897, 2.350055456161499, -0.022903142496943474, -0.4724687933921814, -0.21139825880527496, -0.002736545167863369, 0.03035508468747139, 1.0104498863220215, -1.314164400100708, -0.09826041013002396, 0.3283490240573883, 0.00764821982011199, 0.13574524223804474, -0.060536641627550125, -2.311549425125122, 0.8104051947593689, -0.6055208444595337, -0.03926415368914604, 0.30872491002082825, -0.2877899706363678, -0.13268934190273285, -0.7703224420547485, 1.5612038373947144, 1.4557749032974243, 0.6772529482841492, 1.964799404144287, 0.23975887894630432, -1.4362188577651978, 0.26046136021614075, 1.6715192794799805, 0.024781733751296997, -0.06907099485397339, -1.0857489109039307, -0.0261662770062685, -0.7357661724090576, -0.027889972552657127, 0.44794556498527527, -2.3916478157043457, -0.05298676714301109, 0.06739458441734314, -0.7955579161643982, 0.8425285220146179, 0.13593195378780365, 1.2536113262176514, -1.9898923635482788, -0.25642189383506775, 0.7921743988990784, 0.019964080303907394, -0.2245437502861023, 0.148569718003273, 2.5401625633239746, 1.8504420518875122, 0.5208171606063843, 1.7935837507247925, 0.012402222491800785, 0.008294805884361267, -0.9679484963417053, 0.2032543271780014, -0.1325366497039795, -0.03840230405330658, 1.0281352996826172, 1.1234210729599, -0.22153112292289734, -0.3203416168689728, -0.026892339810729027, -0.6999194622039795, -0.3941859006881714, -0.04994244873523712, -0.04109683632850647, 1.067355990409851, 1.0662997961044312, -0.05276718735694885, -0.1906079798936844, 0.6385494470596313, 0.3400474190711975, -0.09009742736816406, -1.4973585605621338, -1.0590075254440308, -0.6644022464752197, 0.9640104174613953, 0.02958676591515541, -0.05378362908959389, 0.15738023817539215, 0.23224852979183197, 0.32431674003601074, -0.2885059118270874, 0.6558617949485779, 0.09459954500198364, 0.07484888285398483, 1.0217242240905762, -0.9063855409622192, -2.318071126937866, 0.22316999733448029, -0.023709796369075775, 0.5554525852203369, 0.3769572079181671, -0.17429345846176147, 0.9549153447151184, 0.5810475945472717, 1.3133819103240967, 0.7401435375213623, -0.2743420898914337, 0.91603684425354, -0.027801239863038063, -0.6141388416290283, -1.8763248920440674, -0.3762177526950836, -0.01718941703438759, 1.6461164951324463, -0.09705750644207001, -0.19423700869083405, 0.2145950198173523, 0.3236844539642334, 1.7504417896270752, -0.6236075162887573, 0.11801789700984955, 0.2625613808631897, 0.3199675977230072, 0.1748877763748169, -0.042234744876623154, -0.6699957251548767, 1.964171290397644, 0.4531334340572357, 0.5040010809898376, 0.1703653484582901, 0.06608863174915314, -0.037988077849149704, -2.3387765884399414, 0.9425311088562012, 0.2992226779460907, -1.3380396366119385, 0.12509049475193024, -0.36646533012390137, 0.3659362196922302, -4.169164180755615, 0.02718251384794712, -1.029600739479065, -0.16207870841026306, 0.8530747294425964, 1.9973825216293335, -1.4034150838851929, -0.4332062304019928, -0.01442013680934906, -0.011821209453046322, 0.1921420842409134, -0.046300485730171204, 0.06620921939611435, -0.9035196304321289, 1.047440528869629, -0.009401907213032246, 0.20437559485435486, 0.1608000248670578, -0.8392149209976196, -1.6983428001403809, -0.34208783507347107, 0.17714381217956543, -0.008996224030852318, -0.16209562122821808, -2.7167904376983643, -0.25815221667289734, -0.08796685189008713, 0.21341422200202942, 1.410746455192566, -0.4634448289871216, 0.06604685634374619, 0.08711808174848557, 0.02248973771929741, 2.071248769760132, 0.3089877963066101, 0.6861365437507629, -0.42555972933769226, -0.2976662218570709, 1.214791178703308, -0.3108103573322296, 1.1578980684280396, 0.4082230031490326, 0.004215173423290253, -0.1561385840177536, -0.4426388144493103, -0.2983405888080597, 0.16622960567474365, 0.4461000859737396, 0.5304217338562012, -0.17904630303382874, 0.5873302817344666, 1.6470541954040527, -0.010791976004838943, 1.7161811590194702, -0.779769241809845, 0.7395032048225403, -0.8346253037452698, -0.37563666701316833, -0.022325709462165833, -1.0016155242919922, 0.27051717042922974, 1.1385340690612793, -0.03476417809724808, -0.16175588965415955, -1.811380386352539, 2.6515302658081055, 0.6970272660255432, -0.20305317640304565, -0.06302854418754578, 0.965821385383606, 0.9318193793296814, -0.24013955891132355, 0.25007766485214233, 0.05943393334746361, -0.10630691051483154, -1.286358118057251, -1.1243032217025757, 5.345455646514893, -0.2604661285877228, 0.35993024706840515, 0.5036407113075256, -1.164404273033142, -0.015238452702760696, -2.5981884002685547, 0.4066447615623474, -1.697051763534546, 1.7240678071975708, 0.580734133720398, -1.9978013038635254, 0.08039920777082443, 0.0294085294008255, -0.06939227133989334, -0.019850900396704674, -0.3736743927001953, 0.7962198257446289, -0.13260576128959656, -0.14025090634822845, 0.4411383271217346, 1.1195158958435059, 3.4051437377929688, 1.0528513193130493, -1.7710846662521362, -0.6987236142158508, 1.0479049682617188, 0.11957100033760071, -0.01093568466603756, 0.4387497901916504, 0.1355208456516266, -0.8457038998603821, -0.6270774602890015, 0.5842690467834473, 0.08336041122674942, 0.07019267231225967, -0.4862470030784607, -0.6370593905448914, -1.1688401699066162, -1.715556025505066, 1.2447712421417236, -0.22349603474140167, 0.07619783282279968, -1.2334458827972412, -0.09024230390787125, 0.7713375687599182, 0.018837375566363335, 4.2570977210998535, -0.4439297318458557, 0.419127494096756, -0.5179130434989929, -0.4773498475551605, 1.1399312019348145, -2.7115633487701416, -0.7986979484558105, 0.45568913221359253, -0.029846221208572388, 2.223365545272827, -0.9130110144615173, 2.397156000137329, -0.3635631799697876, -0.43541809916496277, -0.832543671131134, 0.15143699944019318, -1.5299649238586426, -0.04729313403367996, 0.31176406145095825, 0.39922577142715454, -2.1563305854797363, 0.18396040797233582, 0.6093851923942566, 0.6484354734420776, -0.19568943977355957, -1.0070421695709229, -0.0335397869348526, 0.7110349535942078, 1.101434588432312, 0.001872133114375174, 0.2630520462989807, 2.0738635063171387, 0.8479275703430176, -0.23293453454971313, -0.08117218315601349, 2.65179181098938, 0.04304341971874237, 0.6998407244682312, 0.11011069267988205, 0.3788638710975647, 0.022396376356482506, 0.010800935328006744, -0.21684128046035767, -0.2949536144733429, 0.02992270700633526, -0.3728284239768982, 0.00017563148867338896, 0.7833088636398315, 0.25790247321128845, 0.7276280522346497, 0.07017984986305237, 0.05864254757761955, 0.12644581496715546, 1.3481600284576416, 0.05092611908912659, 2.155808925628662, 0.9641143083572388, 0.2487451732158661, -0.9279959797859192, -1.3938109874725342, -0.9408072233200073, 0.06704774498939514, 0.4875277876853943, 0.352695107460022, -1.3325716257095337, 0.22655338048934937, 0.4946042597293854, 0.9671459197998047, -1.3660163879394531, 0.5729342699050903, -1.7395007610321045, 0.3884674906730652, 2.259660482406616, -0.0326722078025341, -1.5008704662322998, -0.10728432238101959, -0.4936220049858093, 0.01407638005912304, 0.5196446776390076, 0.21305322647094727, -2.300335168838501, -0.3919525444507599, -0.637191653251648, 0.006551723927259445, 0.06391292810440063, -0.021284712478518486, -0.2640031576156616, 2.608656644821167, 0.1633673459291458, 0.5666257739067078, 0.0862477570772171, -0.1271606683731079, -1.813184380531311, 2.3968019485473633, -0.644687294960022, 0.10251858830451965, -0.004855523817241192, 1.3294403553009033, -0.0874236598610878, -5.4203930631047115e-05, -0.2059800773859024, -2.613678455352783, 0.19905871152877808, -0.18861958384513855, 0.7174434661865234, -1.6834381818771362, 0.0001822003978304565, -0.520860493183136, -0.07739907503128052, -0.015534931793808937, 0.2464200109243393, 0.08131064474582672, -0.08351556956768036, -0.44700172543525696, 0.00472498731687665, 0.7345811128616333, -0.12476561218500137, -1.0971312522888184, 1.9510045051574707, -0.11825870722532272, -1.0501526594161987, -0.44660189747810364, 1.3355580568313599, -0.955504834651947, 2.9061131477355957, 0.05078068748116493, -0.6928927302360535, 0.19106030464172363, -0.3296033442020416, -0.5989875197410583, 0.6024372577667236, 0.22933828830718994, 0.12148900330066681, -0.00597760733217001, -0.41524121165275574, -1.1191540956497192, 0.0018738711951300502, 0.8409499526023865, -1.3619565963745117, 1.1343669891357422, 1.3022063970565796, -0.006063174456357956, -0.38054513931274414, 0.42147698998451233, 0.15172851085662842, 0.6024518609046936, 1.4256538152694702, 2.2072415351867676, 0.4281485974788666, 0.08419936895370483, 0.010546957142651081, -1.1789813041687012, 0.0945853665471077, -0.7645807862281799, -0.5690065622329712, 0.16784074902534485, 1.905730962753296, -1.6481115818023682, -0.03606649860739708, 0.19578194618225098, -0.7876182198524475, 0.029756003990769386, -0.8049914836883545, -2.6879830360412598, -0.3573654890060425, 2.3953309059143066, -3.8801872730255127, -0.2553078234195709, -0.6265513896942139, -0.39715611934661865, -0.7531160116195679, 0.11087741702795029, 0.009034818038344383, -0.0864337757229805, 0.006217299494892359, 0.40727242827415466, 0.6219625473022461, -0.39952611923217773, 0.777084231376648, 0.24704726040363312, -0.26099976897239685, -1.112396001815796, 0.2531746029853821, 0.9592909812927246, -0.174844428896904, 2.675724983215332, 0.6972995400428772, -0.24859188497066498, -0.008137493394315243, 1.5305308103561401, -0.029230520129203796, -0.2414698600769043, -0.024075187742710114, -0.16553305089473724, -0.1546793282032013, -0.04154468700289726, 0.058488793671131134, -0.1207781583070755, 0.08555345237255096, -0.6562536358833313, -0.47005295753479004, 0.6031464338302612, -0.9068046808242798, 0.3049726188182831, -0.23825527727603912, 0.2388787418603897, -0.07721441984176636, -0.391816645860672, 0.8185059428215027, -0.00015975204587448388, 0.0334150455892086, 5.432512283325195, -2.7550649642944336, 0.05425803363323212, -1.096860647201538, -2.244560956954956, 0.014290468767285347, -0.5333300232887268, -2.762387275695801, 0.7739107608795166, 0.21709512174129486, -0.4841453433036804, -0.3668147027492523, 0.9667601585388184, 0.06950000673532486, 0.2720910906791687, 0.34893420338630676, 0.26093482971191406, -0.5568490624427795, 0.12796826660633087, 0.2295847237110138, 0.3355497121810913, -1.628000020980835, -0.035277921706438065, 0.05893354117870331, 0.18184348940849304, 0.06789229810237885, -0.7609049677848816, 0.645272970199585, -0.08565720170736313, 0.13632988929748535, -0.007341747637838125, 0.023479770869016647, -0.35757729411125183, -0.20714743435382843, -4.703641891479492, -1.678563117980957, -0.3355751633644104, 0.4444594979286194, -0.5330247282981873, -0.5995216965675354, -0.20118719339370728, -1.3686622381210327, 1.8604437112808228, 1.0353628396987915, 0.5798287987709045, -0.6436442136764526, -3.053091526031494, 1.1280988454818726, -0.3693860173225403, -0.12511977553367615, -0.09602975100278854, 0.6247042417526245, 0.6675340533256531, 0.3877702057361603, -0.4054821729660034, 3.0861854553222656, 0.0068899947218596935, -0.09946335107088089, -0.18949687480926514, -0.3699173629283905, 0.13023342192173004, -0.15497858822345734, -1.1306160688400269, -0.01580156944692135, -0.0038196512032300234, -1.1825870275497437, -1.4932209253311157, -0.2356501966714859, 0.011548938229680061, -0.1143186166882515, 1.0338760614395142, -0.051450878381729126, -0.21407030522823334, -0.8576698303222656, -0.014205651357769966, 2.4510560035705566, -0.5749978423118591, -0.11391162127256393, -0.25375643372535706, 0.2659975290298462, -0.04375104233622551, -0.6384657025337219, -0.27757251262664795, -0.11767398566007614, 0.6953197717666626, -0.022586753591895103, 0.3313372731208801, -0.4949140250682831, 0.05795690789818764, 0.007597014307975769, 0.41903212666511536, -0.0680646225810051, -0.0036085862666368484, -1.5823479890823364, 1.530606985092163, 0.09166062623262405, -0.15243273973464966, -0.4283387064933777, -0.06032843515276909, 0.005489130038768053, -0.1393776684999466, 0.013236983679234982, 1.9284427165985107, 0.1336962729692459, -0.24015137553215027, 0.1104688048362732, -0.22160761058330536, 0.09435112774372101, 1.2728633880615234, -2.99924898147583, 1.520260214805603, -1.667003870010376, 1.2948529720306396, -0.1801445484161377, -0.09514620155096054, -0.059445567429065704, -0.07976847887039185, 0.1140749529004097, -0.3904137909412384, -0.3178388476371765, 0.30555459856987, -0.019085850566625595, -0.04167715460062027, 0.6770512461662292, -1.2061246633529663, -0.47585421800613403, -3.179293155670166, -0.7750759720802307, -0.20802724361419678, -0.49681735038757324, -0.11207899451255798, -0.3737799823284149, -0.11358292400836945, -0.07006212323904037, 0.04666861519217491, 1.096643090248108, -0.04191054031252861, 1.4699971675872803, -0.03007732704281807, 0.20368953049182892, 0.9089104533195496, 0.8696892857551575, 0.14297309517860413, 0.10153268277645111, 0.4975360929965973, -1.4887657165527344, -1.2482305765151978, 0.024520525708794594, 0.8870295286178589, 0.9542765617370605, 0.20232202112674713, 0.8026859164237976, -0.018765117973089218, -0.06085142120718956, -0.7830354571342468, -0.8678770065307617, 1.1379526853561401, 0.19424758851528168, -1.6845533847808838, -0.7265634536743164, 1.889471411705017, 1.7930880784988403, -0.003673778846859932, -0.929039478302002, 0.009391089901328087, 0.24506568908691406, -0.8653797507286072, 0.049296218901872635, -0.4597665071487427, 0.002082384191453457, 1.0619179010391235, -0.2073706090450287, 0.14127328991889954, 0.15505652129650116, -0.1110777035355568, -0.10794723778963089, -0.0036281857173889875, -0.024806948378682137, -1.946051001548767, 0.12230262160301208, 0.10179144144058228, 0.5262225270271301, 0.0012357502710074186, -0.029623406007885933, -0.37455615401268005, 0.8572292327880859, 0.9175755381584167, -1.971522569656372, -3.635688304901123, 1.3459746837615967, 0.12751264870166779, 0.12293016910552979, 0.3535863161087036, 0.20567364990711212, -0.26502811908721924, 0.04829372093081474, 1.0646308660507202, -0.0003967496450059116, 0.020822253078222275, 0.06534820795059204, 0.06988422572612762, -0.6537546515464783, 0.3031441271305084, -0.649310827255249, -0.11705037951469421, 0.30500730872154236, 0.02866975963115692, -2.150542736053467, 0.534081757068634, -0.0819438099861145, 0.013664238154888153, 0.1656956970691681, 1.672166347503662, 0.07365794479846954, -0.21108323335647583, -1.1447592973709106, 0.006415352690964937, -0.35676929354667664, -0.07434548437595367, -0.042549967765808105, 0.16046547889709473, 0.006954367272555828, 0.3254801332950592, -0.4631485342979431, 1.1550984382629395, -0.4368322193622589, -1.2065480947494507, 0.1098218634724617, -0.13741321861743927, 0.3468492925167084, -0.001063465140759945, -0.1636703759431839, -0.09134887158870697, 0.5032196044921875, -0.40248918533325195, 0.21844759583473206, 0.14034868776798248, 0.03577804192900658, 0.44342488050460815, 0.09314744919538498, -2.088988780975342, -1.0656816959381104, -0.43680691719055176, -1.3527947664260864, 0.17001160979270935, 1.4752154350280762, -1.3352326154708862, 0.06330425292253494, 1.598323106765747, 0.015276283025741577, -0.05106613039970398, 0.10170026123523712, -0.19377021491527557, -0.022340314462780952, -0.5991302132606506, 0.347744345664978, 0.087096206843853, 1.3894908428192139, 0.6794105172157288, -1.1236217021942139, 2.335853338241577, -0.13246791064739227, 0.015254583209753036, -0.03846762329339981, 0.597600519657135, 0.41656872630119324, -0.786578357219696, 0.09743181616067886, -1.266441822052002, -0.8716748356819153, -0.23663318157196045, -2.169341564178467, -0.014455022290349007, -1.076754093170166, -0.2531262934207916, -0.43206220865249634, -3.3052704334259033, -1.797338843345642, 0.8557106256484985, -0.4607604742050171, 1.2637591361999512, -0.7124189734458923, 0.6250210404396057, 1.3971658945083618, -0.08052004128694534, -0.2343287616968155, 0.1868140697479248, 1.575852632522583, -0.4895251393318176, 0.001914925524033606, -0.6505783200263977, 0.8554640412330627, 0.1364307701587677, 1.5954457521438599, 0.0057342336513102055, 1.2303906679153442, 0.27431046962738037, 1.0462111234664917, -1.8813881874084473, -0.05744203180074692, 0.15693488717079163, -0.11882519721984863, -0.006408252753317356, 0.04753255099058151, 0.21722763776779175, -0.5026947259902954, 0.24481064081192017, 0.23673082888126373, 0.5677043795585632, 0.0034032172989100218, 0.5030109286308289, 0.9808173775672913, 0.011992721818387508, -0.1783347874879837, 0.6035495400428772, -0.7119835615158081, 0.44723474979400635, -0.002539522247388959, -0.8561712503433228, 0.7646721005439758, -0.11583751440048218, -0.18541382253170013, 0.08738317340612411, -0.6867271661758423, 0.03617529571056366, -0.9564459919929504, 0.5218488574028015, 0.7479196190834045, 0.00592678040266037, 0.37900981307029724, 1.1270939111709595, 0.42203816771507263, 1.6073575019836426, 0.5173397660255432, 0.1365189254283905, -1.6251299381256104, -1.3389322757720947, 0.1645974963903427, -2.1475303173065186, -0.0046554142609238625, 1.9406954050064087, -0.05503194406628609, -0.026681475341320038, -0.9039844870567322, -0.4721916615962982, 0.14057055115699768, -0.21685737371444702, 0.5404621362686157, 0.7228851914405823, 0.18855372071266174, -0.03724934905767441, -0.6369399428367615, 1.4465677738189697, 1.010861873626709, -0.24287191033363342, -0.6667181253433228, 0.006247915327548981, 0.21482856571674347, -0.2607809603214264, -0.5465633273124695, -1.5058012008666992, 0.182042196393013, -0.5986970067024231, 1.585827112197876, -0.0303414985537529, 1.4852509498596191, 0.566527783870697, -1.1770737171173096, -0.007730710320174694, 1.7294034957885742, 0.06686335057020187, 1.967329978942871, -0.4815848469734192, -1.9398791790008545, -2.1140940189361572, 0.06717737764120102, 0.07218319177627563, 1.4749432802200317, -0.8776227235794067, 0.4317539930343628, -0.5414692759513855, -0.03878292441368103, -0.07757242023944855, -0.037347279489040375, -0.6831848621368408, 0.6686874628067017, -0.2478731870651245, -0.7020112872123718, 0.10667106509208679, 0.12606509029865265, 0.904923677444458, 0.6849212050437927, -0.5167148113250732, -2.4915106296539307, -0.10509927570819855, -0.19453199207782745, 1.6914230585098267, -0.5583532452583313, 2.55582594871521, -0.39555078744888306, -0.715959906578064, 2.5672965049743652, -0.1265924870967865, -0.035570546984672546, -0.51065593957901, 0.26375219225883484, -1.316400170326233, 0.15492655336856842, 0.00550654623657465, 0.01839754357933998, 0.705367922782898, 0.1630033701658249, -0.36023372411727905, 0.08165494352579117, 0.06972159445285797, -0.017690248787403107, -0.591316282749176, 0.38745394349098206, -0.3144371211528778, 0.03765784204006195, -0.08949988335371017, -1.4524874687194824, 0.24595177173614502, -0.010062115266919136, 0.02263568341732025, -0.64481520652771, 0.5060817003250122, -0.9464820623397827, 0.016939722001552582, 2.504983425140381, 0.006206894759088755, -0.316407710313797, -2.5213000774383545, -1.2830535173416138, -1.014400601387024, -0.1762198656797409, -2.0910398960113525, -1.3925554752349854, -0.7680618166923523, 1.2287373542785645, -0.6435154676437378, -0.12402654439210892, -0.057231731712818146, -0.3604836165904999, 0.07754817605018616, 1.9968513250350952, -0.14102862775325775, -1.2229435443878174, 0.4633016288280487, 1.8777098655700684, -0.8226649761199951, 0.9007452726364136, -0.27271878719329834, -0.7006556987762451, -0.35511571168899536, -0.12047581374645233, -0.21848498284816742, -2.7892417907714844, -1.576095700263977, 0.0010978082427754998, 0.15042442083358765, 0.06343051791191101, 0.26642873883247375, 0.4593656063079834, -0.05137735605239868, -0.07629088312387466, -0.8679812550544739, -1.0779221057891846, 0.046434178948402405, -0.8719809055328369, -0.954348087310791, 0.33596333861351013, -0.1863659918308258, -0.8196492791175842, -0.22876018285751343, 0.012841193936765194, -0.8820091485977173, 0.37983810901641846, -0.006562293507158756, 3.015180826187134, -0.14475131034851074, -0.008621704764664173, -3.002410650253296, -0.062389716506004333, -1.0793728828430176, 0.2412157654762268, 0.20216180384159088, -0.2373412549495697, -0.5476242303848267, -1.546504020690918, -0.22817248106002808, 3.672234535217285, 0.720024049282074, -0.7220814228057861, 0.051423393189907074, -0.27157676219940186, -2.1217081546783447, -0.7812145352363586, 1.28836190700531, -0.10942517220973969, 1.0301439762115479, -0.24736106395721436, -0.0334741547703743, -0.015020184218883514, -0.4256104528903961, -0.09520949423313141, -0.5921384692192078, 0.006915531121194363, -0.02162364311516285, -0.27376624941825867, -1.3681446313858032, 0.007373263128101826, 4.689466953277588, 0.05605583265423775, -2.996636390686035, 0.12423759698867798, 1.1624506711959839, 1.138724446296692, 0.22358375787734985, 0.18089640140533447, -0.5770949721336365, -0.023297548294067383, 1.074320673942566, -0.011522882618010044, -0.01318899542093277, 0.6219403147697449, -0.16102996468544006, 0.2156132161617279, 0.06763602048158646, -0.8536797165870667, -0.007848583161830902, -0.4276447594165802, -2.839815378189087, 1.1158000230789185, 0.14277052879333496, -0.22530338168144226, -0.735977828502655, -0.2923493981361389, 0.33669739961624146, 0.3268340826034546, -0.4624682664871216, 0.90828937292099, -1.4206079244613647, -0.20794513821601868, -1.0771795511245728, 0.1998046189546585, -3.3533828258514404, 0.00555017776787281, -0.3637870252132416, 0.2416033148765564, 1.388359546661377, 1.177372932434082, 0.04129638150334358, 0.9324644207954407, 0.3389628529548645, 1.0425437688827515, 0.31367263197898865, 0.5689896941184998, 1.986301064491272, -0.011842919513583183, -1.2159221172332764, -1.7939425706863403, 0.5672372579574585, 0.13739968836307526, -3.0850656032562256, -0.08192653954029083, 0.0503634512424469, -0.08922728151082993, -0.23246820271015167, 0.35298171639442444, 0.7645636200904846, 0.07499245554208755, -0.1593809276819229, 0.021461108699440956, 0.596376359462738, -0.8330013751983643, -0.48249566555023193, 0.6733393669128418, 0.0021704169921576977, 0.5639901161193848, 0.7058627605438232, 0.9355342984199524, -0.11321850121021271, -0.15009824931621552, -0.5839020013809204, -0.10604288429021835, 0.26106396317481995, -1.1367411613464355, -0.08762826770544052, 0.2993408739566803, 0.330552875995636, 1.7068089246749878, -1.4140172004699707, 0.021990947425365448, 0.2223876714706421, -0.34550225734710693, 1.2348089218139648, -4.334140300750732, 1.212339997291565, 0.2519236207008362, -0.5129386782646179, 1.069126844406128, 2.0349650382995605, 0.44609102606773376, -0.5174525380134583, -0.0005563338054344058, 0.384408175945282, -0.03276410698890686, -2.1654675006866455, -0.007808786816895008, 0.6189475655555725, -0.08819194883108139, 0.763947606086731, 0.0798717811703682, -0.5290682315826416, 0.07487497478723526, 0.31759706139564514, 0.01677567884325981, 0.4525712728500366, 0.32144349813461304, -0.04295038804411888, 0.6572943925857544, 0.2004997283220291, 0.3703414797782898, -0.017526492476463318, 0.05666500702500343, -0.09504947811365128, -0.06672722101211548, 0.2854006290435791, -0.09637296944856644, 0.30490127205848694, -0.006495667155832052, 0.35232487320899963, -0.6064922213554382, 1.037634015083313, -0.0958215594291687, 0.24416634440422058, 0.9855993390083313, 2.787525177001953, -0.3976842761039734, -0.27045220136642456, -0.17375816404819489, 0.5243991613388062, 0.12481357157230377, -0.11880844086408615, 0.14167450368404388, -0.0013271740172058344, -0.08756406605243683, 0.7199499011039734, -0.9011307954788208, -0.006702040322124958, -1.523417353630066, -0.06860408931970596, -0.32459789514541626, 0.3851124942302704, -0.39106228947639465, -0.053521160036325455, -0.15356643497943878, -0.3887390196323395, 0.11323697865009308, 0.1300286203622818, -0.9499222636222839, 0.08817684650421143, -0.14437511563301086, -1.0816205739974976, -1.121695637702942, -0.11567056179046631, -0.05865532159805298, -0.8455185294151306, -2.974987506866455, 0.17686636745929718, 1.7322672605514526, 0.3955640196800232, 0.0020978169050067663, 0.1421646922826767, 0.15415866672992706, -3.124119281768799, 0.08361346274614334, -0.3729361295700073, 0.1635027527809143, 0.5877872705459595, 0.01380028948187828, 0.5896522998809814, 1.1539148092269897, 0.5112097859382629, -0.21165408194065094, -0.05249331519007683, -1.748570203781128, -0.24291951954364777, 0.4116966426372528, -0.35625627636909485, 0.9667046070098877, 0.23543107509613037, -0.7884605526924133, -0.4523811638355255, 0.21540336310863495, -0.015453696250915527, 0.298969030380249, 4.433597087860107, 0.9553720951080322, 1.0366281270980835, -1.2690778970718384, 0.49115872383117676, -0.6064012050628662, 0.35337984561920166, -0.05216197296977043, 0.16315323114395142, 0.4852363169193268, -0.11354473233222961, -0.1770460158586502, 0.5387105345726013, 0.936699628829956, 0.27162492275238037, -0.03755892068147659, -0.07355232536792755, -1.9781767129898071, -0.31521740555763245, 0.8052181005477905, 0.5991499423980713, -0.23920616507530212, -0.007244638632982969, 0.13218045234680176, -0.6671668887138367, -1.5882940292358398, -0.12341370433568954, 0.40697529911994934, 0.04201153293251991, 0.34867843985557556, 0.7239015698432922, 0.04973622038960457, 0.00989794172346592, 0.32669100165367126, 0.5416367053985596, 0.4597794711589813, -0.0309834536164999, 1.2008038759231567, -0.3678104281425476, -0.20633462071418762, 0.10909868776798248, 0.4717651307582855, -0.15511822700500488, 0.04396367073059082, 0.24366989731788635, 0.19700437784194946, -0.13575468957424164, -0.6026970148086548, 0.01840025931596756, -0.0023873988538980484, 1.2539737224578857, 0.4558027982711792, 0.07741550356149673, -0.13100630044937134, -0.019091177731752396, -0.06585288792848587, 0.00178545992821455, 2.8406105041503906, 0.8717990517616272, -0.8839459419250488, 1.089942455291748, 0.3617059588432312, -1.2124236822128296, 0.13006041944026947, -0.48508331179618835, 0.1650213599205017, 1.495678424835205, 0.12577007710933685, -0.26172322034835815, 0.3443377912044525, 0.08407678455114365, -1.6035763025283813, 0.6441558599472046, -0.4770960807800293, 0.39315006136894226, -0.17436344921588898, 0.28225842118263245, -2.34725284576416, 0.002182746771723032, -0.9610452055931091, 0.7517293691635132, 0.2865782678127289, 0.35141289234161377, -0.16213162243366241, 1.084120512008667, 1.3289998769760132, 0.6808276772499084, -0.44382572174072266, 0.04311222955584526, 0.2603885531425476, -0.30370673537254333, 0.34597310423851013, -1.2284528017044067, 0.22604206204414368, 4.066803455352783, -0.10902852565050125, -0.6494734287261963, -0.3109586834907532, 1.3642340898513794, 0.6678835153579712, 0.19842174649238586, -0.15916430950164795, 0.9161461591720581, -0.96025550365448, 0.35891270637512207, -0.7124581933021545, -0.0143081434071064, 0.025409655645489693, -1.6725239753723145, 0.09844639897346497, 0.6112960577011108, 0.0009767801966518164, -0.3203992545604706, 1.810670018196106, -0.11083447933197021, -0.30712276697158813, -1.3109735250473022, -0.387007474899292, 0.08607923239469528, 0.1818126142024994, 0.014162893407046795, 0.1850259155035019, -1.566578984260559, 1.2390989065170288, 0.7491368055343628, 1.0468918085098267, -0.09190140664577484, -0.0376182422041893, 2.5881128311157227, -0.07396999001502991, 0.1639520674943924, 0.029037097468972206, 0.5350874662399292, 1.7556285858154297, 0.05872549116611481, -0.02839493937790394, 0.2706863582134247, 1.0013890266418457, 0.09279663115739822, 0.08924587815999985, -0.6744868159294128, 0.5777208805084229, 0.48093247413635254, 2.0466320514678955, 1.6197993755340576, 1.606247067451477, -0.21965759992599487, -0.04085438698530197, -0.37155115604400635, -0.18989965319633484, -0.9289425015449524, -0.6444960236549377, -0.06803078949451447, -2.6220502853393555, -0.28193482756614685, 0.05234750360250473, -0.02501797489821911, -0.6069162487983704, -0.3103281259536743, 1.0643028020858765, 0.1157640889286995, 0.22289007902145386, -1.4274542331695557, -0.40557342767715454, 0.2862531244754791, -0.2775295674800873, -1.8730816841125488, 0.002453829627484083, -0.3253176808357239, -0.8322291970252991, 0.928470253944397, -0.083708755671978, -0.05974655970931053, 0.0717805027961731, -0.4085502028465271, 0.054056260734796524, -0.05771341919898987, -0.8356977701187134, 0.04715004935860634, -2.4109129905700684, -0.5022290945053101, 1.0465590953826904, 0.09168050438165665, -2.973361015319824, -0.4306098520755768, -0.8616775274276733, 0.5976358652114868, 0.7943874597549438, -0.8824875354766846, 0.0632205680012703, -0.2970416247844696, 1.3503516912460327, 1.3182621002197266, 0.35673272609710693, -0.7748146653175354, -0.07207611203193665, 0.21190676093101501, -0.03902464359998703, -0.3984910547733307, -0.009891039691865444, 2.0238265991210938, -0.37649816274642944, -0.012528176419436932, 0.051885053515434265, 0.45618686079978943, 0.193094402551651, -2.7406489849090576, -2.310457229614258, 3.1090657711029053, -0.2513289749622345, 0.32208648324012756, 0.3069673776626587, 1.1262949705123901, 3.0284600257873535, 0.3089725971221924, 0.9991046190261841, 0.7778383493423462, -2.5140817165374756, -0.7522740364074707, 0.41952845454216003, 0.07568850368261337, 0.37595030665397644, 0.31042563915252686, 0.3653023838996887, -0.9957144856452942, -0.02213824912905693, -0.3074491024017334, 0.0055388775654137135, 0.0823940634727478, -0.5082171559333801, -0.262889564037323, -0.3969022333621979, 0.12289289385080338, -0.1340196132659912, 0.07577049732208252, 0.16088365018367767, -0.6928479671478271, 0.770839512348175, -0.8944894075393677, -0.022383714094758034, 0.37077680230140686, -1.2669999599456787, 0.2820078730583191, 0.28172412514686584, -0.7506877779960632, 2.7324907779693604, -0.5867559313774109, -2.2298684120178223, 0.1266590803861618, 0.13582883775234222, 0.9706752300262451, -0.763759970664978, 0.5521143674850464, 0.03156939893960953, 0.33393794298171997, 0.11667253077030182, -0.03289921581745148, -0.3258504867553711, 0.5222841501235962, -1.4107352495193481, -0.20619481801986694, 0.6704626083374023, -0.7393364310264587, -0.023579899221658707, -0.014442432671785355, 1.0470356941223145, 0.10527089983224869, 1.0393668413162231, 0.8140026330947876, 0.5604146718978882, -0.8966383337974548, -0.019841261208057404, 0.8603562712669373, 0.3728196918964386, 0.25122469663619995, -0.6686595678329468, 0.21827206015586853, -0.017419692128896713, -4.6789045333862305, -0.8854994177818298, 1.5565587282180786, 0.5674816370010376, 0.1988813728094101, 1.8516459465026855, 0.5929645299911499, -1.0746369361877441, 0.9823911786079407, -0.9633512496948242, 1.177802324295044, 0.2778223752975464, 0.23524509370326996, 0.17030033469200134, -0.01173369400203228, -0.02791411057114601, -0.5614296197891235, -0.029526932165026665, -0.1080535426735878, -0.9693272709846497, -0.09395292401313782, -0.24998106062412262, 0.22274979948997498, -0.15849731862545013, 0.21604475378990173, 1.3697257041931152, 1.6895179748535156, -0.9367718696594238, -0.10024022310972214, -0.3974815309047699, 0.6750622987747192, -0.39125242829322815, 0.0182600487023592, -2.1668529510498047, -0.20044414699077606, -0.2645568251609802, -1.4446860551834106, -0.041610993444919586, 0.995637059211731, 0.3362456262111664, -0.02435300126671791, 2.0663602352142334, -0.08125743269920349, -0.2501909136772156, 1.4688321352005005, -0.07883594185113907, -2.5938422679901123, 0.06836992502212524, -0.3696819543838501, -0.1476038545370102, -0.01683058962225914, 0.006320183631032705, -0.441152423620224, -0.7737961411476135, -0.31688350439071655, 0.023371806368231773, -0.5816278457641602, 0.606121838092804, 1.1908830404281616, -1.2977778911590576, 0.3091062307357788, -0.12227346003055573, 0.26973971724510193, -0.2060079425573349, -0.030067963525652885, -0.14806537330150604, 0.04481487721204758, 0.6375369429588318, 1.0002586841583252, -0.0010174618801102042, 0.011098113842308521, 0.3752352297306061, 0.27201905846595764, -0.714721143245697, -0.2100895345211029, 0.8397740125656128, 2.318345785140991, 0.503926157951355, 0.11513207107782364, -0.0003259899967815727, 0.2851150333881378, -0.2579779326915741, 0.3489362299442291, 1.8308684825897217, -0.18158122897148132, -0.002081987913697958, 0.9214709997177124, 1.2653108835220337, 0.5574736595153809, 0.8722929358482361, -2.1771812438964844, -1.6450682878494263, 0.12639102339744568, 0.12074597924947739, 1.2086619138717651, 1.5806421041488647, -0.2089904397726059, -1.2667597532272339, -2.3622395992279053, -0.3198852837085724, -0.018792787566781044, -1.2551140785217285, 0.1739244908094406, -0.02149275876581669, -0.7238407731056213, -1.2653145790100098, -0.8159995675086975, -0.6669460535049438, -0.33889955282211304, 0.3328961730003357, 0.7394288778305054, 0.392184317111969, 2.1248016357421875, -0.9829556345939636, 0.18863381445407867, -0.759958803653717, -0.0785234272480011, -0.5489240288734436, 0.7825670838356018, 0.08007681369781494, 0.07842087000608444, 1.1576497554779053, -0.0003325722645968199, 0.5366728901863098, 0.009605953469872475, -0.35010552406311035, -0.4193071722984314, 1.6498756408691406, -0.23716498911380768, 0.28744256496429443, -0.22749872505664825, -0.053810276091098785, 0.16320958733558655, -0.11432037502527237, -0.06874439865350723, -0.5002853274345398, -0.760862410068512, -1.1042872667312622, -0.013952909968793392, -0.338364839553833, -0.3039332330226898, 0.5585339069366455, 2.9100825786590576, 0.051856376230716705, -0.15358750522136688, 0.06284995377063751, -1.472252607345581, 0.03989566117525101, 0.3116157054901123, 0.7991815209388733, 1.2998656034469604, -0.07337823510169983, 0.09214320033788681, 0.42466965317726135, 0.613490104675293, 1.3705244064331055, -0.010649267584085464, -0.29100239276885986, -0.2858419716358185, -0.16747747361660004, -0.47909149527549744, -0.027004778385162354, 0.7589807510375977, -0.1700400710105896, 0.47120076417922974, -0.3291470408439636, 0.4621541202068329, -0.31744053959846497, -1.0464402437210083, -0.5545775294303894, -0.15560707449913025, -0.9760631322860718, -0.6398593783378601, 0.8989179730415344, 0.07382041215896606, -0.37654799222946167, -0.8197934031486511, 0.32024240493774414, -1.1741642951965332, 0.389734148979187, 1.0769727230072021, 1.3784277439117432, 1.550668478012085, 0.2567436993122101, 0.24162495136260986, 2.0772476196289062, -0.3916887044906616, -2.4336116313934326, 0.0947943925857544, -1.4973946809768677, -0.2964839041233063, 1.4327361583709717, -0.07727877050638199, 0.7084039449691772, 0.42198532819747925, 1.6193820238113403, 0.4228569567203522, 1.3190735578536987, -0.0154213672503829, -1.520175814628601, -0.16396783292293549, 0.0016722391592338681, -0.6942600607872009, -0.2642904222011566, -1.003313660621643, 0.4219316244125366, -0.5495429039001465, -0.7665550112724304, 0.3130573332309723, 0.38906556367874146, -0.023570101708173752, -0.003095806809142232, 0.19973738491535187, -0.4744519591331482, -0.15079912543296814, 0.024655086919665337, -0.20422570407390594, -1.1865226030349731, -0.31670910120010376, 0.13500696420669556, 0.305912584066391, 0.23403120040893555, -0.605343222618103, -0.22540228068828583, -0.06599845737218857, -0.7207051515579224, 1.1343601942062378, -0.04392243176698685, -0.04658369719982147, 0.17464405298233032, -0.028023583814501762, 0.034957267343997955, 0.8055605888366699, -0.6913548707962036, 0.7231597900390625, 0.49136030673980713, 0.36339616775512695, 0.816944420337677, 0.2955351769924164, 3.977215528488159, 2.793731927871704, -0.05349680781364441, -5.288466930389404, 0.3746362030506134, -1.425502061843872, -1.6690657138824463, -0.07642503082752228, -0.7021234631538391, -0.47945261001586914, -1.161624550819397, -0.040682584047317505, 0.07206772267818451, 0.036977265030145645, -1.3034406900405884, 0.9264380931854248, 0.2858657240867615, -0.272585391998291, 0.18104523420333862, -0.4469509422779083, 0.05556909739971161, 0.8959177136421204, 0.0008257174049504101, -0.4186784029006958, 2.8012473583221436, 1.989547848701477, -0.06173168122768402, -0.498919278383255, 0.06256691366434097, 0.11681056022644043, 0.1319998800754547, -0.5162876844406128, 0.18057364225387573, -0.2236492782831192, 0.5060031414031982, -0.17285749316215515, 0.0055994270369410515, 0.03450831398367882, 0.21986308693885803, 0.05815626680850983, -0.38680848479270935, 0.40548253059387207, 0.035165172070264816, -0.8265571594238281, 0.09622221440076828, -1.972359299659729, -0.07877785712480545, 1.231305480003357, 0.12051030993461609, 0.942547619342804, 0.653973400592804, 0.4817301332950592, 0.46720409393310547, -0.21477915346622467, 2.0461699962615967, 1.0169482231140137, -0.527226448059082, -0.02359510399401188, 1.2465869188308716, -0.4440084397792816, -0.0801420584321022, 0.30559834837913513, 0.04659558832645416, 0.08034878224134445, 0.1795888990163803, -0.1198294535279274, 0.058453530073165894, -0.35424932837486267, 0.3610159158706665, 0.3010517954826355, -1.6403409242630005, 3.9174180030822754, 1.1560752391815186, 3.5017263889312744, -1.574331283569336, 1.8430126905441284, -0.2598722577095032, 0.14789943397045135, 1.1986894607543945, -0.16785651445388794, 0.706867516040802, 0.04846230521798134, 0.47291576862335205, -0.09620936959981918, -1.8391937017440796, 0.0689668357372284, -0.10997781902551651, 0.1796731948852539, 1.061699628829956, 0.06985990703105927, -1.1666696071624756, -1.0453705787658691, 0.35616669058799744, 1.2215659618377686, -0.5779476761817932, 0.3275066614151001, -0.11245189607143402, -0.3638056814670563, 0.2582744061946869, -0.09732440114021301, -0.6809355616569519, 0.00584565382450819, 0.19736745953559875, 0.4975108504295349, 1.6479190587997437, 0.8927769660949707, 0.63318932056427, -0.5031448602676392, -0.04738082364201546, 0.44469401240348816, 0.8936607241630554, -0.4071211814880371, -1.2585761547088623, 0.07371296733617783, -0.9948007464408875, -0.4671734869480133, -0.023892344906926155, -0.3160291612148285, 0.4412868916988373, 0.1602024883031845, 0.016002502292394638, -0.679038405418396, 0.5158773064613342, 0.32545337080955505, 0.03489883244037628, -0.27738165855407715, -1.3709241151809692, -0.18857072293758392, 0.8724060654640198, 0.2725668251514435, -1.4392155408859253, -0.04011236131191254, 0.23489417135715485, 0.031019486486911774, 0.177835151553154, -0.5030509829521179, 0.2215229719877243, -0.007895756512880325, -0.16866806149482727, -0.2684253454208374, 1.61955726146698, 0.05319368466734886, 0.41859740018844604, 0.9660966396331787, 0.13156069815158844, 0.5454710125923157, -0.08583953231573105, 0.5994940996170044, 1.3515146970748901, 0.13716337084770203, -0.6965458393096924, 0.5761277079582214, 0.41408273577690125, -0.027270713821053505, -0.15265192091464996, -0.04977300018072128, 0.1167069599032402, 0.7087017893791199, -0.630905032157898, 0.2089444100856781, 0.22665831446647644, 0.23507261276245117, -0.24333040416240692, 1.0954853296279907, 0.11593051254749298, -0.01611998863518238, -0.224791020154953, 0.20106473565101624, -0.036093346774578094, -0.35232996940612793, 1.26162588596344, 0.035936661064624786, 0.08145636320114136, 0.24287985265254974, 1.7520849704742432, 1.4609699249267578, -0.8824825882911682, 0.39392200112342834, -1.1960488557815552, -0.67978835105896, 0.2703387439250946, -2.482330322265625, 1.4385244846343994, -0.13372403383255005, 0.2173260748386383, 0.08620911836624146, 0.014785791747272015, 0.1287483125925064, -1.0376018285751343, 0.029379840940237045, 0.1452108472585678, 0.11746226996183395, 0.3895101845264435, 0.6285138130187988, -6.8211750984191895, -0.5497004389762878, 0.026531068608164787, 2.46297287940979, -0.14433406293392181, -0.7416054606437683, -0.6914013028144836, -0.5244441628456116, -0.08349476009607315, -0.7026543021202087, 0.9278864860534668, 0.2095227837562561, -2.727360725402832, 0.09582769870758057, 0.7965359091758728, -1.0745424032211304, 0.42372235655784607, -0.18904653191566467, -0.491569846868515, -0.06463652849197388, -0.21956588327884674, 0.05831250548362732, -0.03737335652112961, -0.9721866846084595, 0.17296864092350006, -0.548552930355072, 0.28523799777030945, -0.2852058410644531, 3.1928763389587402, 0.053471408784389496, 1.0513097047805786, -2.344332695007324, 0.5210999846458435, -0.7890102863311768, -0.3739754259586334, 0.08878224343061447, -0.01744280382990837, -0.6915842294692993, -0.7810602784156799, 0.013290726579725742, 0.23089371621608734, 0.9369817972183228, 0.07947325706481934, -0.21394005417823792, -0.9809824228286743, 1.8121213912963867, 0.7899482250213623, -0.017771190032362938, -0.079452745616436, -0.01711426116526127, -5.442511558532715, -0.05260736867785454, -0.9876968860626221, 0.29984909296035767, 0.2622586190700531, 0.23661518096923828, -2.058805465698242, 0.16866523027420044, -1.2705528736114502, 0.5845854878425598, 0.2427765280008316, -0.09676244109869003, 1.024976134300232, 0.1256287842988968, -0.0017269247910007834, -0.7345821261405945, 0.14416083693504333, 0.21037420630455017, 0.29045823216438293, -0.3863425850868225, -0.27923059463500977, -0.744221568107605, 1.804785132408142, -0.3896048963069916, 0.17919433116912842, 0.43507733941078186, -1.1750012636184692, -2.028107166290283, -0.5526493191719055, -0.6083717942237854, 0.4906123876571655, 1.2367032766342163, 0.09390144050121307, -0.24106408655643463, -1.0705174207687378, 0.45693808794021606, 0.3140923082828522, -0.5813726782798767, -1.713352918624878, -2.4360556602478027, -0.9334383606910706, 0.021719327196478844, 0.8780122995376587, -0.1408347338438034, -0.37527093291282654, -0.235882967710495, -0.0027164791245013475, -3.628532886505127, -1.412436604499817, 1.674060583114624, -1.8427852392196655, -0.6279311776161194, -1.4623830318450928, 0.325815349817276, 0.0499027818441391, 0.08596333861351013, -1.2579197883605957, 0.010050827637314796, 0.48372212052345276, -0.026256520301103592, 0.329384446144104, 0.48706743121147156, 0.7210198044776917, -1.0437387228012085, -0.18141557276248932, 0.10630573332309723, 1.9858839511871338, 0.8635672330856323, 0.2527090609073639, -0.26067501306533813, 0.33492526412010193, 3.564249038696289, 0.8967402577400208, -2.7226929664611816, 0.2767728269100189, 0.0029625017195940018, 0.025142595171928406, -0.073625847697258, 0.5038086771965027, -0.059233397245407104, 3.074747323989868, 1.1755567789077759, -0.29999616742134094, -0.011339444667100906, 1.2929757833480835, 0.10790543258190155, -0.39112144708633423, 0.2365557849407196, -1.266701579093933, 1.554307222366333, -0.09185921400785446, 0.105128712952137, 0.048758409917354584, -2.042231321334839, -0.3680192530155182, -1.5708835124969482, -0.49864423274993896, 0.2119964063167572, -0.9729159474372864, 0.32206016778945923, -0.7165108323097229, -0.23201048374176025, -1.2277566194534302, 0.8961262702941895, -0.6548042893409729, 0.0558963306248188, 0.0892459899187088, 1.4146887063980103, -0.31196582317352295, -0.0058549619279801846, -0.7004655003547668, -1.3423620462417603, -0.6872274279594421, -1.249611496925354, -0.6262360215187073, -0.9397925138473511, -0.027970487251877785, -1.3378793001174927, 0.07733845710754395, -0.4231768548488617, -0.7642861604690552, -0.013623600825667381, 0.6731215119361877, -0.12129451334476471, -2.083129405975342, -0.23589830100536346, -0.8079003691673279, -0.04240412637591362, -0.5893880128860474, 0.10970479995012283, -0.11994346976280212, -0.45612189173698425, 0.31959810853004456, -2.2710461616516113, 0.082719586789608, 0.06991598010063171, -0.5493207573890686, -0.08086541295051575, 0.17522309720516205, 0.6664440035820007, 0.4247296452522278, 1.021536111831665, 0.488839328289032, 1.29990816116333, 0.6183924674987793, 8.629839897155762, -0.1912374496459961, 1.2172043323516846, 1.2241548299789429, -0.5103113651275635, -1.8411091566085815, -0.1883281171321869, 0.27506914734840393, -0.26053574681282043, 0.4937533736228943, -0.17757897078990936, 1.0676298141479492, 1.8265501260757446, 0.28230345249176025, 3.6112217903137207, 0.49568289518356323, -0.02440706081688404, 0.5893341302871704, 0.18661892414093018, -0.2830507457256317, -0.5557411313056946, -0.16319207847118378, -2.027970552444458, 0.021882236003875732, 0.39689987897872925, 0.012746517546474934, 0.39873605966567993, 0.17680613696575165, 0.11656970530748367, 1.1380633115768433, 0.5324829816818237, -0.49625855684280396, 0.15453669428825378, 0.016132770106196404, -0.3731457591056824, 0.7161712646484375, 0.2250705510377884, 0.1783226728439331, -0.0546284057199955, -1.216943383216858, 0.6498258709907532, 0.29510968923568726, 1.1250241994857788, -1.085496187210083, 0.0721152126789093, 0.42078927159309387, 0.2163313776254654, -0.02748904377222061, 0.3897455632686615, -0.04482471942901611, -0.01729552261531353, 0.9154868721961975, -0.8226101994514465, -0.7356134057044983, 0.7472152709960938, -0.05902145430445671, -0.08989430218935013, 0.07990701496601105, -2.340756893157959, 0.6894562244415283, -0.43653151392936707, -0.2451195865869522, 1.8358711004257202, -0.4781082570552826, 2.405487298965454, -1.6214311122894287, 0.3631347119808197, -0.47114700078964233, 1.0779751539230347, -0.45032456517219543, -1.65487539768219, -0.09674292057752609, 0.29696062207221985, 0.11540821939706802, 1.1710069179534912, 1.588510513305664, 0.18307538330554962, 0.7332441210746765, -0.8795585632324219, 0.33149340748786926, 0.11720788478851318, 0.5193460583686829, 0.9585278034210205, 0.4022524952888489, 0.9216886758804321, -0.05868125706911087, 1.0433522462844849, 0.5672264099121094, -0.3887555003166199, 0.3601740002632141, -0.6241777539253235, -0.2848915159702301, 1.2179783582687378, -0.17488528788089752, 1.2056066989898682, -0.06306841224431992, 0.4140246510505676, 1.0473685264587402, -0.08949368447065353, -0.28215429186820984, 0.3544456362724304, 0.033072106540203094, 0.10627452284097672, 0.7053720951080322, -1.925330400466919, -3.6598849296569824, -2.498635768890381, 0.06578926742076874, 0.2626540958881378, -0.5533071160316467, 0.062372006475925446, -0.35764360427856445, 1.2060012817382812, 0.12025924772024155, -0.830341100692749, -0.02106260508298874, -0.0008974994416348636, 0.18401220440864563, 0.0382930152118206, -1.4403589963912964, 1.0575274229049683, 0.0009288862347602844, 0.45956864953041077, -0.7966741323471069, -0.02121925912797451, 0.03564983978867531, 0.9924540519714355, -0.3166165053844452, -0.02786480262875557, -0.06748497486114502, -0.0949954017996788, 0.4182043969631195, 0.0005069726030342281, -0.03141374513506889, -0.03389160707592964, 0.22206389904022217, -0.5955411791801453, -0.037194784730672836, -0.17292843759059906, -0.3058587908744812, -0.4085236191749573, 0.30230045318603516, -0.8521987795829773, 0.8381003141403198, -0.10557755827903748, 1.070162057876587, -0.0148802874609828, -0.2505682110786438, 0.4333434998989105, -0.535283088684082, -0.14828424155712128, -0.47795411944389343, 0.049809277057647705, -0.19684971868991852, -0.09535891562700272, -0.3082789480686188, 0.09433729946613312, 0.10437571257352829, 0.168450728058815, 0.8801714181900024, 0.4832221567630768, 3.2646164894104004, 0.3048303425312042, -2.5339605808258057, 0.22638922929763794, 0.4454694390296936, 2.150522232055664, 2.3752481937408447, 0.12307997792959213, -0.6702562570571899, -0.11598733812570572, 0.3650750517845154, -0.5311475992202759, -1.5804024934768677, 0.4657338559627533, -0.19188356399536133, -0.20179592072963715, -0.9191611409187317, 0.08836876600980759, 2.179412603378296, 0.009099810384213924, -0.5880391001701355, 0.06983266025781631, 0.2986043393611908, -0.10803920030593872, 0.3418513536453247, -1.1866304874420166, 2.6911137104034424, 0.300725519657135, -0.07864492386579514, -0.003642726456746459, -1.4356510639190674, 0.19754402339458466, -0.4554991126060486, 0.41426849365234375, -0.943188488483429, 0.6236861348152161, 1.319683313369751, 0.09187925606966019, -0.011223613284528255, -1.4433155059814453, -0.03527217358350754, -0.02944551408290863, -0.26030582189559937, -0.1390167623758316, 0.6925979256629944, -0.9289072751998901, -1.0436159372329712, -0.5499486923217773, -1.321601390838623, -1.377157211303711, 0.06907740980386734, -0.9022441506385803, -0.039107225835323334, -1.0054281949996948, -0.5376176238059998, 0.1228574588894844, 1.1278067827224731, -1.314989447593689, -1.0123121738433838, -0.6814673542976379, -0.3546311557292938, 0.08031405508518219, 0.025052599608898163, -0.7329369783401489, 0.28631356358528137, -0.09735772013664246, 0.1871938556432724, 0.8448624610900879, -0.866508424282074, -0.39955708384513855, -0.9821110367774963, -0.9940415620803833, 0.0666780173778534, 2.5860586166381836, 0.35874179005622864, 0.5748124718666077, -0.3646535873413086, -0.00445934571325779, -0.3182244896888733, 0.39620307087898254, 0.13937288522720337, -0.5572690963745117, 0.2979141175746918, -1.5216898918151855, -0.10238122940063477, 0.06148697808384895, 1.5166947841644287, -0.11760485917329788, -0.03323632851243019, -1.0319279432296753, 0.6004177927970886, 0.7759510278701782, 0.7912452816963196, -0.418682336807251, -0.40449246764183044, -1.5166372060775757, 1.1589534282684326, -0.5632322430610657, 0.007158249616622925, -0.023101823404431343, -0.6502147912979126, 1.8327018022537231, -0.46746161580085754, 0.8052079677581787, 1.16275954246521, -0.5376355051994324, -0.024383969604969025, 0.6451564431190491, -0.32394441962242126, 1.5408660173416138, 0.4585903584957123, 0.051722098141908646, -0.188978910446167, -0.9512898921966553, -0.3001682162284851, -1.652401328086853, -0.5073557496070862, 0.6784002780914307, 0.3278186321258545, -0.10559050738811493, 0.6486464738845825, 0.13421036303043365, -0.7404187321662903, -0.03961596265435219, -0.553256630897522, 2.315411329269409, 0.1561310738325119, 0.09005656838417053, -0.0447692908346653, 0.2410655915737152, 0.5670223236083984, 0.022138170897960663, -0.7361032962799072, 0.05177405849099159, 0.1752869188785553, 0.09115047007799149, -0.17342761158943176, 0.14953093230724335, 0.21177543699741364, -0.019520709291100502, 0.20097477734088898, -0.1481982171535492, 0.8646490573883057, -0.09794145822525024, 0.808790922164917, 0.07208652049303055, 0.39680662751197815, -0.4496636986732483, 1.8625949621200562, 0.3115125298500061, -0.01059765461832285, -0.7419338822364807, -0.6278995275497437, -0.3905729353427887, 0.024241693317890167, -1.4375320672988892, 0.05381574109196663, 0.07231263816356659, -0.18784454464912415, -0.009733939543366432, 0.35722818970680237, -0.322408527135849, 0.05482149496674538, -0.034568339586257935, 2.465477228164673, 0.38650304079055786, -0.2591835856437683, 1.5548603534698486, 2.309941291809082, 0.19602231681346893, -0.23848702013492584, 0.9969576001167297, 0.37770652770996094, 0.0956881195306778, -0.1275945007801056, 0.4091968536376953, 1.5105211734771729, 0.09186650812625885, 0.15492181479930878, -1.6710857152938843, 0.014128141105175018, -0.162405326962471, -0.5066013932228088, 0.02094930224120617, -1.2866426706314087, -0.033119380474090576, 0.204460009932518, -0.21812571585178375, 1.7105647325515747, 0.12051743268966675, 0.4627462327480316, 0.2893098294734955, -0.2887539565563202, 1.69278085231781, -1.3795390129089355, -0.6217060089111328, 0.47186189889907837, -0.856884241104126, -0.2227986603975296, -0.09851551800966263, 0.012112339027225971, 0.31049972772598267, -0.2239401638507843, 4.156124114990234, -0.3282313644886017, 1.0597355365753174, 0.09907037764787674, 0.011813336983323097, -0.8394528031349182, 0.5428811311721802, -0.836707353591919, 0.4672832787036896, -0.07897715270519257, 0.8964241743087769, -1.173435926437378, 0.14815814793109894, -0.03247739374637604, -0.301411509513855, 0.1919703334569931, 0.03864239528775215, 0.8983923196792603, -0.034398969262838364, -0.2774137556552887, -0.078263059258461, 0.06230609863996506, 0.33463796973228455, 4.112137317657471, 0.0716053768992424, 0.7103090882301331, 0.46252351999282837, 1.7557275295257568, -0.11269635707139969, 1.8931849002838135, -0.3322734534740448, 0.9511815309524536, 0.008226236328482628, 1.3607901334762573, 1.488936424255371, 0.03440551459789276, 0.5034951567649841, -0.08994670957326889, 0.8590577840805054, 0.06668113172054291, -1.1677167415618896, 2.0265142917633057, 1.2413413524627686, 0.2674546241760254, 0.32275232672691345, 0.046577613800764084, -1.5701931715011597, 2.770779246930033e-05, 1.607109785079956, -0.001580838463269174, 1.2315243482589722, 3.688814640045166, -0.18142639100551605, -1.5945465564727783, 0.586102306842804, 1.0890671014785767, 0.9409343600273132, -1.8883965015411377, -0.6467821598052979, -0.6066218614578247, -0.05415978655219078, -0.7531421184539795, 0.3250538408756256, 2.957960367202759, 0.14973874390125275, -0.0432983934879303, 0.056597478687763214, 0.00956837646663189, -0.7690398693084717, 0.4958503544330597, -0.046438492834568024, 3.95536470413208, -0.809032678604126, 0.3954220414161682, -0.37327465415000916, -0.04514029622077942, -0.5226719975471497, -0.001562342164106667, 0.2898583710193634, 0.9425382614135742, -0.5366148352622986, 0.5568166375160217, -0.06421099603176117, 0.4204683005809784, 0.42732757329940796, -0.17510396242141724, -0.5169117450714111, 0.028317326679825783, -0.21779470145702362, 0.17621515691280365, 0.05568915233016014, -0.20634090900421143, 0.43849942088127136, -0.503501296043396, 0.07472793757915497, 0.0753437876701355, -0.7060930728912354, -0.6242038607597351, -0.032422665506601334, 0.030608193948864937, 1.1995458602905273, -0.04752056300640106, 0.4460870325565338, -0.2508745491504669, -0.078098826110363, -0.021584469825029373, 0.1055656373500824, -0.45621293783187866, 5.168583869934082, 1.0326855182647705, 0.17074862122535706, -0.0021374886855483055, -0.4774346947669983, -0.1250496804714203, 0.5583054423332214, 0.2684672772884369, -0.45958176255226135, 1.1106504201889038, 2.5795130729675293, 0.028851626440882683, -0.7483913898468018, 2.0091776847839355, -0.14616400003433228, 0.002391860354691744, -0.5879307389259338, -0.27356940507888794, -0.468546062707901, -0.004951317794620991, 3.763615131378174, -0.07890751957893372, -0.3280714750289917, -0.7120571732521057, -0.38287973403930664, -0.30823618173599243, -1.8161215782165527, 0.07968778908252716, -0.20624181628227234, 0.2989128828048706, -0.15695001184940338, 0.5828138589859009, 0.05246836319565773, 1.1580500602722168, 0.1533551961183548, 0.6070585250854492, 0.05434396490454674, -0.2262071967124939, 0.014288950711488724, -0.5132731199264526, 0.3531079888343811, 0.19014737010002136, 0.0455314964056015, -0.02570122852921486, -0.1133708581328392, 1.9036977291107178, 0.22523033618927002, 0.7474486231803894, 0.4636434316635132, -0.6726996898651123, 0.22005806863307953, -1.4416794776916504, -0.17602796852588654, -0.04162779450416565, -0.9706844687461853, 0.23292836546897888, -0.7238492965698242, -0.46930041909217834, -0.07114271819591522, 0.0202620942145586, -0.7310059666633606, -2.3172972202301025, -0.3942372500896454, -2.081772565841675, -0.7333164215087891, -0.0021120819728821516, -0.38387489318847656, -0.08740835636854172, 0.9506090879440308, 0.17966189980506897, 0.3976300358772278, 2.309206008911133, -0.7359554171562195, -0.02713332511484623, 0.8655624389648438, -0.45434463024139404, -0.5524678826332092, 2.4256420135498047, -0.2189655900001526, -0.5063206553459167, 0.21153931319713593, -0.03522542864084244, 0.8480477929115295, -1.9339702129364014, -0.003354765707626939, 0.5129984021186829, 0.3431363105773926, -0.6268961429595947, 0.34974557161331177, 0.25922414660453796, 0.8157529234886169, 0.21056298911571503, 0.0235595740377903, 0.8520249724388123, -1.3177697658538818, 0.012477228417992592, -0.8132563829421997, 0.2516617178916931, 0.020888211205601692, 0.9818295836448669, 0.1425309181213379, 0.06731213629245758, 1.2583028078079224, 1.1408506631851196, 0.19112394750118256, 0.1239699274301529, -1.3455531597137451, -0.008348025381565094, 1.1282466650009155, -0.6290122866630554, -0.268004447221756, 0.696265459060669, 0.21991583704948425, 0.0575709342956543, 0.9663020372390747, -0.12449292838573456, -0.005209791474044323, 0.40146172046661377, -0.4167707562446594, -0.06547581404447556, -0.15382200479507446, 0.5090857744216919, 0.07010219246149063, 0.22700458765029907, -0.04121122881770134, -0.0471080057322979, 0.8001016974449158, 0.0877172127366066, 0.9142138361930847, 0.5084325671195984, -0.19897785782814026, 1.4099617004394531, -0.002516016596928239, 0.014649547636508942, -0.9497814178466797, 0.6815964579582214, 0.12620189785957336, 1.9818356037139893, -1.0908732414245605, 0.006627980154007673, 0.9864757061004639, -0.07924049347639084, 1.3462380170822144, 2.2658658027648926, 0.2635030150413513, 0.24228531122207642, 0.0035345633514225483, -0.021970879286527634, 1.6040735244750977, -0.1844957023859024, 0.033433035016059875, -1.5226428508758545, -0.005025327671319246, 0.9149078726768494, 0.06125403568148613, -1.7107446193695068, 0.37436193227767944, -1.073654294013977, 2.896150588989258, -0.004701373632997274, -0.3100992739200592, 0.007217272184789181, 0.0743856355547905, -1.5540506839752197, 1.7534493207931519, -0.5879983305931091, 0.005426757037639618, -0.03326058387756348, 0.16896946728229523, 1.679713487625122, 0.8587313294410706, 0.6828444600105286, -0.13367509841918945, 0.0639970526099205, 1.503861665725708, -0.28374794125556946, 0.6051574349403381, 2.4009251594543457, 0.5490602850914001, 0.32779383659362793, -0.6915571093559265, 0.5064041018486023, 0.2569333612918854, -0.08213038742542267, -0.05758204683661461, 0.011469234712421894, 0.33517202734947205, -0.2232014536857605, -0.003302802564576268, -0.19647054374217987, -1.091831088066101, 0.3011532723903656, -0.6030229330062866, 1.1222705841064453, -0.8454676270484924, 1.22614324092865, 0.5764548778533936, 0.011909371241927147, 0.41091999411582947, -0.4734632074832916, -0.012324564158916473, 0.29407185316085815, 0.6971942782402039, 0.10120360553264618, -0.2763480842113495, 0.05271052196621895, 0.12030227482318878, 0.05906057357788086, 1.1840119361877441, -0.021104464307427406, -0.9476203322410583, -0.43612509965896606, 0.8441850543022156, 0.3694957196712494, -0.22945767641067505, -0.06457352638244629, -0.8111536502838135, -0.012446142733097076, -0.12361549586057663, -0.015268196351826191, 0.6470354199409485, 0.14402663707733154, -2.329092264175415, -0.5224061608314514, 1.6731661558151245, -0.681283712387085, -1.453055739402771, 1.481939673423767, -0.06821967661380768, -0.4719422459602356, 0.9977652430534363, 0.41847410798072815, 0.9160252809524536, -0.19793210923671722, 0.5748463869094849, -0.5293205976486206, 0.9636369347572327, 0.1018533855676651, -0.6075423955917358, 0.2039133459329605, -0.20789103209972382, 0.060342736542224884, 2.363609552383423, 1.544614315032959, 0.009321143850684166, -0.6790251731872559, 0.1213773861527443, -0.10813874751329422, 1.1838172674179077, -0.1425178498029709, 0.03047081083059311, 0.2555992603302002, -0.21108072996139526, -1.1069166660308838, -0.11786767095327377, -0.2364056557416916, 0.15615031123161316, -1.3195698261260986, -0.5002294182777405, -0.07425298541784286, -0.7491316795349121, 1.0265886783599854, -0.018081821501255035, -1.4618514776229858, 0.47451213002204895, 0.48932528495788574, -0.11676938086748123, 0.6466103792190552, -0.24091185629367828, -0.4250231981277466, -0.10749351233243942, -0.35569658875465393, -0.6637768149375916, -3.7498843669891357, -0.48876193165779114, 0.703361988067627, -0.020061802119016647, 1.6653820276260376, 2.4710066318511963, 0.2747243344783783, -0.1832062304019928, 0.3396001160144806, -0.6078705191612244, 0.008269315585494041, -0.5634012818336487, 0.2141525149345398, -1.193184733390808, 0.31976383924484253, 0.9614468812942505, 0.6168308854103088, 0.5250428915023804, -1.477589726448059, 0.08099334686994553, -1.4020401239395142, -0.7664691209793091, -0.05852088704705238, -0.11171181499958038, -2.7421295642852783, -0.2954011857509613, -1.6279722452163696, 0.20602618157863617, 1.5793453454971313, -0.36972519755363464, -0.13178735971450806, 0.22610200941562653, 0.5345532298088074, 0.19307567179203033, 0.13543538749217987, -1.159680724143982, 1.486488699913025, 0.030155718326568604, 0.047907985746860504, 0.7725028395652771, 1.1908550262451172, -0.0403473936021328, 0.054433368146419525, 0.02343244105577469, -5.838366508483887, 1.2825043201446533, 0.16427670419216156, -0.2638890743255615, -0.12244618684053421, -2.5607128143310547, 1.9648281335830688, 0.04323526471853256, 2.25431752204895, 0.03693327307701111, -0.08833232522010803, 1.3195959329605103, 0.07392917573451996, -0.15057916939258575, -0.5147498846054077, -1.0027586221694946, 1.2215040922164917, 0.11460954695940018, -0.2225046157836914, 0.2544758915901184, -1.201194405555725, 1.870808482170105, 0.06599074602127075, 0.006834907922893763, 1.99755859375, 0.791280210018158, 0.20395362377166748, 0.14170505106449127, -0.20011502504348755, -0.0973651111125946, 1.2139174938201904, -0.43585121631622314, -0.34784168004989624, -0.07219754159450531, -0.3127698004245758, 0.07659312337636948, 1.0486772060394287, 0.9098833203315735, 0.051223818212747574, -0.0054849134758114815, -0.4459921717643738, -0.7216646671295166, -0.8306698799133301, 0.8988598585128784, 1.4479081630706787, -1.0650079250335693, 0.9061330556869507, 1.0544397830963135, 0.5064160227775574, 0.5826927423477173, -0.13744020462036133, 0.4096883237361908, -1.5553067922592163, 1.2822185754776, 0.19319969415664673, -0.02362598478794098, -0.020990736782550812, 0.06053468585014343, 0.08943825960159302, 1.0696417093276978, 2.044940233230591, 0.7438734769821167, 0.04608229920268059, -0.0302002914249897, -0.2231820970773697, -0.029396921396255493, 0.8446165919303894, -1.0834870338439941, -0.9113448262214661, 0.10169622302055359, -0.3920164108276367, -0.07542682439088821, -0.2782916724681854, -1.347549557685852, -0.06316477060317993, 0.051265668123960495, 0.11508076637983322, -0.527442216873169, 0.21088670194149017, 0.053749438375234604, -0.31549903750419617, 0.2480846345424652, 1.659833550453186, -1.617228627204895, 2.7951595783233643, -0.07647432386875153, -0.5223448276519775, -0.40404680371284485, -0.07531549036502838, 0.40991517901420593, -0.6741324067115784, 0.04526960477232933, -1.0545177459716797, 1.278737187385559, 0.9512331485748291, -1.496146559715271, -1.2670360803604126, -1.6367524862289429, 1.4718945026397705, 0.7195892333984375, 0.32638245820999146, -0.9009097814559937, 0.32753705978393555, 1.0648396015167236, -0.36624860763549805, -0.0001378512679366395, 0.2793098986148834, 0.09768471121788025, 1.47348153591156, 0.01223282516002655, 0.9352662563323975, 1.3532614707946777, -0.552277147769928, 0.3764632046222687, 0.02050214633345604, 1.5794436931610107, -1.0981107950210571, 0.04054873064160347, 0.10810039192438126, 0.19044598937034607, -0.049068283289670944, 0.8000030517578125, -0.47829505801200867, 0.03942231088876724, -0.07769692689180374, 1.5590217113494873, 1.0402922630310059, -0.0015280855586752295, 0.5055751204490662, -0.1363970786333084, 0.69515061378479, 0.03147848695516586, -0.23327891528606415, -0.32121795415878296, -0.025549359619617462, 1.4102632999420166, 0.2970016300678253, 0.4615318477153778, 0.17669175565242767, 0.003666275879368186, 1.7419224977493286, 0.0056612794287502766, -1.110673189163208, -2.0631046295166016, 0.1035849079489708, -0.3166780471801758, 0.1314404010772705, -0.06670025736093521, 0.52558833360672, -0.2431943416595459, 1.606730341911316, -0.8477272987365723, 1.915573000907898, 3.0307955741882324, 0.20107904076576233, 0.009562384337186813, 1.1326255798339844, -0.8763004541397095, 0.00770584587007761, 1.0139057636260986, -0.05939917266368866, -0.05965300276875496, -0.035143930464982986, -0.050194043666124344, 0.1261681765317917, -0.1480245441198349, 0.7631819844245911, -0.18777133524417877, -0.07302023470401764, -0.3889867961406708, -0.6441379189491272, -1.354854941368103, -1.4073786735534668, 1.7935856580734253, 0.24100960791110992, 0.6709762811660767, 0.023699847981333733, 1.8350778818130493, -0.586310863494873, -0.17133048176765442, 0.07867864519357681, -0.6626299619674683, -0.13140320777893066, -0.05246146023273468, -0.5967714190483093, -0.23352962732315063, -0.21307720243930817, -0.10122493654489517, 0.18041586875915527, 1.009673833847046, 0.21151161193847656, 0.6548015475273132, 1.2423855066299438, -0.5299474000930786, 0.13079127669334412, -0.017516808584332466, 5.036468029022217, -0.5893369913101196, -3.8406167030334473, -0.06763259321451187, -0.7451350688934326, 0.23158524930477142, 0.1263808161020279, 0.41119518876075745, 0.4266926050186157, -0.6869640350341797, 0.829279899597168, -1.0155225992202759, 0.8249505162239075, -1.4211117029190063, -2.6971349716186523, -0.10788886249065399, -0.39813441038131714, -0.8625025153160095, -0.3760243058204651, 1.3094348907470703, 0.20562425255775452, -0.035167284309864044, 0.32702258229255676, 0.21390360593795776, 0.15537290275096893, 0.02192278578877449, -0.3154542148113251, -0.6018688082695007, 0.6032227873802185, -0.31321418285369873, 0.11365437507629395, -0.7666806578636169, 1.4123705625534058, 0.04704595357179642, -0.13085871934890747, -0.40479543805122375, 0.00013622046390082687, -0.9638141989707947, 1.1539751291275024, -1.0215444564819336, 0.3729448616504669, 0.4455893635749817, 0.9983122944831848, -0.6664651036262512, -0.8285035490989685, -0.0013135692570358515, 0.01057969406247139, -1.0658091306686401, -0.21183021366596222, 3.04044508934021, -0.11366930603981018, 0.35600680112838745, -0.046681445091962814, -1.0181128978729248, -0.6953798532485962, -0.944283127784729, 1.9824814796447754, -0.005105461925268173, 0.2249438315629959, 0.02912917174398899, -0.010948091745376587, 0.399598628282547, 0.6217949390411377, 0.11639164388179779, -4.222588062286377, -0.22849276661872864, -0.6453916430473328, 0.20599380135536194, -0.0768575444817543, 1.3184821605682373, 0.7039487957954407, 0.5744127035140991, -1.3492599725723267, -0.20759962499141693, -1.6351706981658936, -0.15294897556304932, 0.24334222078323364, -0.758098304271698, 2.72406005859375, 0.7591153383255005, -0.11273237317800522, 0.14987073838710785, 0.3161960542201996, -0.37463870644569397, -1.0281795263290405, -0.008580319583415985, -0.48101234436035156, 0.9041489362716675, 0.178836390376091, 0.14428164064884186, -0.10997031629085541, -0.9328101277351379, -0.4114360511302948, 0.030490579083561897, 0.03558868169784546, -0.6335639953613281, 0.24542644619941711, 0.3020381033420563, 1.731873631477356, 0.6735380291938782, 0.11089696735143661, -0.0670332983136177, 0.16323620080947876, -0.7303518652915955, 0.37227529287338257, -2.911195993423462, 0.7105481624603271, -0.27662894129753113, 1.2970088720321655, 1.1252084970474243, 0.2460373342037201, 0.9971840381622314, -0.5272720456123352, -0.24719254672527313, 0.10699276626110077, 0.04349807649850845, 0.4697324335575104, -0.02042471058666706, 0.07287832349538803, 0.2932336628437042, 1.8991948366165161, 1.0664399862289429, 0.026700088754296303, -0.23477604985237122, -1.037716031074524, -1.9558131694793701, 0.059604715555906296, -0.21806974709033966, 0.01795792207121849, -0.7486191391944885, 0.01857151836156845, -0.4965837001800537, -0.2696007788181305, -0.21064570546150208, -0.006467124447226524, 0.6344201564788818, -2.2889091968536377, -0.17139436304569244, -1.6313787698745728, -0.10973504930734634, -1.2619123458862305, -0.09691574424505234, 0.7016304731369019, -1.581007957458496, -0.1507132649421692, -0.4980485141277313, -0.5756947994232178, 2.1151909828186035, -0.07743848860263824, 0.041711412370204926, 0.2989502251148224, -2.286458730697632, 0.04852405935525894, 0.9952967166900635, 0.6662386655807495, -0.0632568970322609, 0.15818406641483307, -0.12674498558044434, 0.6196152567863464, -0.03445431590080261, -1.077648639678955, 1.3491220474243164, -0.010110920295119286, -0.08496424555778503, -0.09643618017435074, -0.8560361862182617, 0.28181564807891846, -0.04784649610519409, 0.24974310398101807, -0.21577320992946625, 0.006163578014820814, 0.2917884588241577, -0.12296324223279953, 0.1604156643152237, -1.6211292743682861, -0.22601862251758575, 1.2641593217849731, -0.7268698811531067, 0.04660047963261604, 0.43186676502227783, -0.5750274658203125, 0.10821712017059326, 1.2281041145324707, -1.7468527555465698, 1.6895946264266968, 0.06966474652290344, -0.3183416724205017, -0.5320609211921692, -1.0670222043991089, 0.011460804380476475, 0.5648279190063477, -1.0960674285888672, 0.30442583560943604, 0.21408911049365997, -0.5050064325332642, -0.8841216564178467, -0.03888338431715965, 0.3106340765953064, 0.09824100881814957, -0.016571706160902977, -0.4972447156906128, 0.17678391933441162, -1.1697510480880737, 0.26689088344573975, -0.14890120923519135, -0.6803250312805176, 1.109784722328186, -1.0870914459228516, 1.2940772771835327, 0.44094428420066833, 0.38501664996147156, 1.4052683115005493, -0.35098546743392944, 0.6587824821472168, 0.8660949468612671, -0.03899895399808884, 1.1370429992675781, 0.04074044153094292, -0.10460357367992401, 0.00825488194823265, 0.23195475339889526, -0.36017316579818726, 2.608175039291382, 0.17286966741085052, 0.9905714988708496, 0.3701155483722687, 0.2630688548088074, -0.7537053823471069, -0.665606677532196, 0.49715548753738403, -0.035175397992134094, 0.15358306467533112, 0.11696992814540863, -0.4726312458515167, 0.042235199362039566, 0.4278803765773773, 0.31275272369384766, 0.15845562517642975, -0.1484762281179428, -0.21211086213588715, -0.3268418312072754, 0.05912581831216812, 0.006509014870971441, 0.06022726744413376, -0.8511369228363037, -0.47285863757133484, 0.6252162456512451, 0.3085925281047821, -0.7443095445632935, -1.1107654571533203, 1.2757489681243896, -0.3926582336425781, -0.7316681742668152, 0.026358509436249733, -0.22699585556983948, 0.09477602690458298, -0.17986106872558594, -0.992414653301239, 0.580517053604126, -1.0733246803283691, 0.003997956868261099, 0.8532119989395142, 3.0040860176086426, -1.1412204504013062, -0.09347384423017502, 2.9056296348571777, 0.4771477282047272, -0.17451581358909607, -0.03729451447725296, -0.10296282172203064, 0.2931500971317291, -0.8520000576972961, -0.11914557218551636, 2.2336490154266357, -0.011109059676527977, 0.5521547794342041, 0.014940198510885239, 0.3900171220302582, 0.10335291922092438, -0.7665214538574219, 0.30659693479537964, -0.0024124549236148596, -0.33461126685142517, 1.238484263420105, -0.0013972262386232615, -0.03984440863132477, -0.5055978298187256, -0.5535370707511902, -0.09037070721387863, 1.608145833015442, 0.31626731157302856, 0.054662592709064484, 0.07590814679861069, 0.9946457147598267, 0.6445339322090149, -0.034729477018117905, 0.41762420535087585, 0.984555721282959, 0.8854657411575317, -0.026131579652428627, 0.029643869027495384, -0.3299647867679596, -0.06253471225500107, 0.04956349357962608, -1.4253251552581787, -0.30306291580200195, -0.25327959656715393, -0.04744090139865875, -0.5212945938110352, 0.008763693273067474, -0.23969128727912903, 0.05968932807445526, -0.4058849811553955, -0.013975548557937145, 0.8844277858734131, -0.034697454422712326, 0.0972321555018425, -1.0804357528686523, 3.3741185665130615, 0.16776247322559357, 0.3705332279205322, -0.39205169677734375, -0.5411168932914734, -1.6408754587173462, 0.2472912073135376, 0.04339876398444176, 0.838175892829895, 1.3222593069076538, 1.2614418268203735, -0.056696679443120956, -0.9785911440849304, -0.04908474534749985, -0.43781980872154236, -0.49201786518096924, 0.34084680676460266, 0.08079579472541809, -0.20709766447544098, -0.0025471881963312626, -0.9061000943183899, -3.928204298019409, 0.2722070813179016, 1.1093509197235107, -0.338918000459671, -0.9479494690895081, -0.11198195815086365, 0.0012966177891939878, -0.13442952930927277, -0.02157915011048317, -0.11449502408504486, -0.5729612708091736, 0.04432768002152443, -0.565571129322052, -0.33791351318359375, 0.4772000312805176, -0.9185150265693665, 0.47048845887184143, 0.9754906296730042, -1.202426552772522, -0.42366546392440796, -2.66371488571167, 0.34199026226997375, 0.02492578886449337, -2.8511319160461426, 1.4355645179748535, -1.297533631324768, 0.4241396486759186, 0.326084166765213, 0.5067719221115112, -0.0373520590364933, -0.6007593870162964, -1.7255125045776367, 0.3031623661518097, -0.19370333850383759, 0.5411208271980286, 3.872979164123535, -0.05158384144306183, -0.005415812134742737, 0.3230692744255066, -0.4970872104167938, 0.07339870929718018, 0.9933555722236633, 1.668369174003601, -1.8873652219772339, 0.15828289091587067, -0.17398595809936523, 0.6032604575157166, -0.2817424237728119, 0.3158363699913025, -0.12700311839580536, 0.15067526698112488, 0.019008450210094452, 0.34752318263053894, -0.025817349553108215, 0.32378703355789185, 0.3481518030166626, -1.2031257152557373, -0.20442548394203186, 0.2186504304409027, 0.006959930993616581, -0.02647589147090912, -0.5198571681976318, -0.3497626483440399, -0.0045614042319357395, -1.3814650774002075, -1.1372982263565063, 0.04678703472018242, -0.9464577436447144, -0.16391101479530334, -2.355738878250122, -1.6139110326766968, 0.1421308070421219, 0.20452392101287842, -1.4827687740325928, 0.2787119448184967, 0.0167184229940176, 0.35190796852111816, 0.2737694978713989, 0.4104919135570526, 1.672729730606079, 0.9269072413444519, -0.1157122552394867, -0.18719661235809326, 7.358056545257568, -0.09288511425256729, -1.348050594329834, 0.302710622549057, -0.09156767278909683, -0.10032187402248383, -0.8811042904853821, 0.15225253999233246, -0.28732410073280334, 0.02662934921681881, 0.5647814273834229, -1.3688647747039795, 1.3336693048477173, -0.003052887972444296, -0.07304604351520538, 0.005097254179418087, -0.10728014260530472, 0.5083937644958496, 2.024418592453003, 0.19839173555374146, 1.0870742797851562, -0.0969298928976059, 3.2910475730895996, -0.10332159698009491, -0.1321619749069214, -0.44280216097831726, 0.09190516173839569, 0.032361797988414764, 0.015411196276545525, -0.6017141342163086, -0.4792545735836029, 1.2006456851959229, 0.47213539481163025, -1.2508426904678345, -3.0383262634277344, 0.09211017936468124, -3.237412929534912, 0.29089459776878357, 0.2239942103624344, -1.0746995210647583, -0.01733466424047947, -1.099800705909729, -0.05420888215303421, 0.7504366636276245, 0.05161954462528229, -0.3692481219768524, -0.31962811946868896, 1.0128167867660522, -0.30985498428344727, -0.7222456932067871, 0.010881765745580196, -0.3011004626750946, 0.9954097867012024, -1.870700478553772, 1.0612552165985107, 0.7141228318214417, 0.970276415348053, 0.1745644360780716, 1.0511549711227417, -2.515717029571533, 0.052658695727586746, -0.031326696276664734, 0.3374386429786682, -0.26198291778564453, 0.3435850739479065, 0.009400099515914917, 0.431697279214859, -1.0260204076766968, -0.004549558740109205, 0.33232632279396057, 0.006188954226672649, -0.447774201631546, 0.02166951261460781, -0.1421491801738739, 1.3494369983673096, -0.04080679640173912, 0.36703598499298096, 0.12142526358366013, 0.19426316022872925, 0.5220734477043152, -0.13900737464427948, 1.8633817434310913, 0.06332206726074219, 0.03108530305325985, -0.6594700813293457, -0.0866287499666214, 0.5714097619056702, 0.31749069690704346, -0.48162296414375305, 0.10687646269798279, -2.3139660358428955, 1.7112042903900146, -0.4060713052749634, 0.46260786056518555, -3.136152744293213, -1.1233187913894653, -1.6704901456832886, -1.047468900680542, -0.36787015199661255, -0.36451834440231323, -4.161149501800537, -0.2817515432834625, -1.6139270067214966, -0.29333096742630005, -0.32259514927864075, 1.5691806077957153, -0.01932615414261818, -0.782645046710968, 1.1995691061019897, -0.1240907832980156, 0.5368875861167908, 0.0679166167974472, 0.11488883197307587, -0.3173855245113373, -0.008392921648919582, 0.1554509401321411, 0.31141141057014465, -0.40964677929878235, -0.13442401587963104, 0.08311429619789124, 0.71445631980896, 0.45919543504714966, 0.3693510591983795, -0.07348167151212692, -3.065521001815796, 0.02960159443318844, 0.2576959431171417, 0.4375685453414917, 0.18165290355682373, 0.07949987053871155, 0.4038311541080475, -0.0164139736443758, 1.225359320640564, -2.9737727642059326, -0.07244084775447845, 0.016753563657402992, 0.6173047423362732, -0.8480647802352905, -0.7191691398620605, 0.45964503288269043, -1.3919332027435303, -0.243827685713768, -1.0512217283248901, -1.0944925546646118, 0.04952950030565262, 0.6124967336654663, 0.5372917056083679, -1.005677342414856, -0.27889031171798706, 0.20537099242210388, -1.6539506912231445, 2.357795476913452, 1.2391713857650757, 0.05519244819879532, 0.7496261596679688, -1.056786298751831, 0.010178404860198498, 0.8876028060913086, -1.6996256113052368, 0.5189957022666931, -0.1635926514863968, 1.1341197490692139, 0.8031358122825623, -0.012299311347305775, -0.49041780829429626, 0.9327625036239624, 0.809245228767395, 0.8481518626213074, 0.1213170662522316, 1.275524377822876, 0.6507323980331421, -0.5907420516014099, 0.3126503825187683, 0.22170598804950714, -0.04867178201675415, -0.09539826959371567, 0.8550459742546082, -0.9173860549926758, 1.0868674516677856, 2.5306944847106934, 0.06348371505737305, 0.2693963348865509, -0.8166859149932861, -0.09954845905303955, 1.7592577934265137, 0.10596352815628052, 1.7999829053878784, -0.01864142157137394, 1.6391239166259766, -0.24078170955181122, -0.4618786573410034, 0.5520063042640686, 0.022650044411420822, -1.7809600830078125, -0.20967014133930206, -0.4609314799308777, 0.08220183104276657, -0.05391588434576988, 0.5404088497161865, -0.18354050815105438, 0.1066223755478859, 0.20706601440906525, 0.1910553127527237, -0.01577925682067871, -0.269463449716568, 0.303528755903244, 0.29308903217315674, 0.5340688824653625, -0.004263543523848057, 2.1216142177581787, 0.3991183042526245, 0.47461041808128357, 0.07648567855358124, -0.027035027742385864, 0.021696731448173523, -1.4267975091934204, 0.1236400455236435, -0.20468442142009735, 1.3784408569335938, 0.36712950468063354, -1.7421637773513794, -0.4003598690032959, -0.8827211856842041, -0.48967692255973816, 0.43595123291015625, -0.01255207508802414, -0.20371708273887634, -0.4190681576728821, -0.4279758930206299, -1.9534409046173096, 0.01669551245868206, 0.17815449833869934, 1.6055153608322144, -0.003948793280869722, -3.784306287765503, -0.1973833441734314, -0.22299647331237793, -0.9664966464042664, 1.9017544984817505, 0.051465217024087906, 1.6425460577011108, -0.8425114750862122, 0.6902065277099609, -0.43609774112701416, 0.04385661706328392, 0.6584835052490234, -0.028529275208711624, -0.8604002594947815, 0.9268150329589844, 0.25820812582969666, -4.049580097198486, 1.600629210472107, 0.8768835067749023, -0.011360013857483864, -0.630375862121582, -1.05258309841156, 1.5648025274276733, -0.23864376544952393, -0.9221183657646179, 0.6972793340682983, -0.8192995190620422, 0.8061820864677429, 0.7893768548965454, -0.4182467758655548, -0.10471110045909882, -0.0771176666021347, 0.5627137422561646, 0.8722782731056213, 0.30352315306663513, -1.3023277521133423, -0.12054099887609482, -0.07176530361175537, -0.5377950072288513, -0.01848558709025383, 0.48311614990234375, 0.5042104721069336, 0.024589603766798973, -0.34631669521331787, -0.1452137976884842, -1.2714383602142334, 0.26759880781173706, 0.8209999203681946, 0.3702114224433899, -0.016651013866066933, -0.6456007361412048, -0.5341681838035583, 0.20371873676776886, 0.07630348205566406, 0.2573240101337433, -0.6454942226409912, -0.014197571203112602, 1.7474102973937988, -1.1606647968292236, -0.2708514332771301, 2.289679765701294, 0.024206269532442093, 0.36180442571640015, -1.8402395248413086, -0.3990601599216461, -1.7261888980865479, 1.3326654434204102, -0.7189326882362366, 0.09743864834308624, -0.14717063307762146, 0.09896355867385864, 0.5568608045578003, -0.8705146908760071, 0.03453558310866356, -2.672760486602783, 0.5773114562034607, -0.3469863831996918, -0.19597464799880981, -0.27770504355430603, -0.016258087009191513, -0.050292834639549255, -0.8482988476753235, -0.013984530232846737, 0.08255381137132645, -0.28180205821990967, 1.8460296392440796, -0.38094744086265564, 3.2815473079681396, 0.5771579742431641, 0.583061695098877, -1.2821000814437866, -1.4432824850082397, 0.11507093161344528, 0.24519512057304382, -0.08314092457294464, -0.559885561466217, -0.13100208342075348, -0.07600542157888412, 0.3288423418998718, -0.23160825669765472, 0.10569975525140762, -0.41645538806915283, 0.053198669105768204, 0.4361109733581543, -0.17823505401611328, -0.02724265307188034, 0.11117463558912277, -0.2568240463733673, 0.8881747722625732, 0.08993497490882874, -1.2501646280288696, 0.09242087602615356, 0.15145859122276306, 0.3613301217556, -0.09297499060630798, 1.10377836227417, 0.16904997825622559, -0.18860597908496857, -2.2041029930114746, -1.047448754310608, 0.6838607788085938, 0.24649976193904877, 0.43570685386657715, 0.05660666897892952, 0.8065125346183777, 0.14356909692287445, 0.11616799980401993, 0.696081280708313, -0.9968844652175903, -0.3005533814430237, -2.6122686862945557, -0.4192117750644684, 0.03700091689825058, -1.3808223009109497, -0.1641320288181305, -0.5349820256233215, -0.2677716016769409, 0.3365841507911682, -0.8080571889877319, 0.20646058022975922, -0.018385443836450577, 0.19533754885196686, -1.1499019861221313, 0.04317335784435272, 1.5625101327896118, 0.19029031693935394, 0.12444203346967697, 0.3720099925994873, -0.3958352208137512, 0.07006261497735977, 0.005012683570384979, -2.6563937664031982, 0.407246470451355, -0.3135785460472107, -0.6165405511856079, -0.8506835103034973, 0.9923016428947449, 0.4942224323749542, -0.6367149949073792, -0.6849439740180969, 3.1094038486480713, -0.25770536065101624, 0.0025177558418363333, -0.4258246421813965, 0.7816759347915649, 0.5204548835754395, -0.2255004346370697, -1.2894790172576904, 0.27636873722076416, -0.8628138303756714, -1.6877392530441284, -0.003915269859135151, 0.002745550824329257, -0.21442532539367676, 0.8671702146530151, 0.4915635585784912, 0.5695474743843079, -0.07486914098262787, 0.9863035082817078, 1.3155754804611206, 1.0808055400848389, -0.11996714770793915, 0.3042161762714386, -0.36655479669570923, 0.43609362840652466, 0.2580607235431671, 0.05128178745508194, -0.2613625228404999, 0.5461325645446777, -0.6126792430877686, -0.06656870990991592, -0.8919636607170105, 0.45008307695388794, -0.025072429329156876, 0.36885693669319153, 1.0777019262313843, 2.128404140472412, -0.38406914472579956, -0.41975414752960205, -1.014611840248108, -0.2590152621269226, -0.2245696485042572, 1.0842205286026, 0.9728299975395203, -0.3019748330116272, -0.06248379126191139, 1.4151235818862915, -0.07875948399305344, 2.993842840194702, -0.20549601316452026, 0.18790081143379211, -0.6313639879226685, -0.2604869604110718, 0.17873981595039368, -0.9286357760429382, -0.5113248229026794, 0.18083733320236206, 0.834104061126709, 0.15732674300670624, -1.078270673751831, -0.16498194634914398, -0.5978586673736572, 0.9060832262039185, 0.4893689751625061, -1.3202192783355713, -0.4057258367538452, 0.23881691694259644, 1.7943613529205322, -0.28907671570777893, 1.44673490524292, 0.049533139914274216, 2.774050712585449, -1.1493691205978394, 1.591689944267273, 1.1293643712997437, -0.06350715458393097, -0.004651722963899374, -0.06982599943876266, -0.522517740726471, 1.9737837314605713, 0.1985001116991043, -2.4411122798919678, 0.22872011363506317, 0.12115772068500519, -0.12224076688289642, 0.2813929319381714, -1.8392751216888428, -0.08741874992847443, -0.183443084359169, 1.1260792016983032, -0.27405500411987305, 0.0766608864068985, 0.08925532549619675, -1.4970159530639648, -0.3110896944999695, 1.8327604532241821, -0.04309738799929619, -0.927361786365509, -0.39602503180503845, 0.06280170381069183, -0.00810309499502182, -1.6521222591400146, 0.3347322940826416, -0.03177854046225548, -0.10528665781021118, 0.17898491024971008, 0.7750967144966125, -0.19802431762218475, 0.3320898413658142, -0.1627294272184372, 0.626035213470459, -1.4085561037063599, -0.3974483907222748, -0.21249832212924957, 1.1999493837356567, 0.8965655565261841, -0.414890319108963, 0.07478994131088257, -0.7973495721817017, 0.03293474763631821, 0.1119106337428093, -0.24584321677684784, 0.023687070235610008, 0.026909159496426582, 1.489179253578186, -2.616046667098999, 0.39872729778289795], [0.13692551851272583, 1.9295796155929565, 0.011088998056948185, 1.503887414932251, -0.19828610122203827, -1.7288850545883179, 0.0004570306045934558, 1.435672402381897, 0.37168624997138977, 3.805577516555786, -1.3949804306030273, -1.5600874423980713, -0.15592226386070251, -0.12309642881155014, 0.15327678620815277, -0.3181822896003723, -0.09811969846487045, -0.29297465085983276, -0.5005366206169128, 0.15704622864723206, -1.3753643035888672, 0.5109446048736572, -0.024357600137591362, -0.8301783800125122, 0.07909857481718063, -0.34127873182296753, 0.028393104672431946, 0.4535457193851471, 1.3330819606781006, -0.004158358089625835, -0.5405697226524353, -0.40608689188957214, 0.6251235008239746, -0.5791827440261841, 0.012056331150233746, -0.49936428666114807, 0.534615159034729, 0.09964083135128021, 0.8184765577316284, -1.249832034111023, 0.7559388279914856, -0.08100351691246033, -0.02201835811138153, 0.12792213261127472, -0.4640111029148102, -0.613243579864502, 0.005356105510145426, 0.4796435236930847, 0.48449215292930603, 0.1466665416955948, -0.46994227170944214, 0.1722676157951355, -0.06866144388914108, -0.22017337381839752, -0.1346406787633896, 0.32859402894973755, 0.042990542948246, 0.0011764069786295295, -0.04948408901691437, 0.02363310009241104, 0.31390199065208435, 0.05488164350390434, -0.2469726800918579, 0.4569593071937561, 0.8061264753341675, -0.020405111834406853, 0.09125538170337677, 2.7089316844940186, -0.19619683921337128, 0.31687048077583313, 0.04797644168138504, -0.926260232925415, -0.07635137438774109, 1.8662539720535278, -0.5143628120422363, -0.11808566749095917, -0.6653826236724854, 1.602110505104065, 0.0008680993923917413, -0.7492629885673523, -0.01465226337313652, -0.4594350755214691, -0.27354973554611206, -0.16509447991847992, 0.3194494843482971, 0.41506898403167725, -0.06107550114393234, 0.7325522303581238, -0.31957805156707764, 0.5204652547836304, 0.7892449498176575, 0.22192810475826263, -0.011297778226435184, 0.2517571747303009, -1.1878831386566162, -1.4472105503082275, 0.5162184834480286, -0.4722934365272522, -0.16865092515945435, -0.11986525356769562, 0.04208531975746155, -0.0028991892468184233, -0.17579255998134613, 0.1664263755083084, 0.05027182400226593, -0.0054777939803898335, -0.16868938505649567, -0.11525938659906387, 0.8969796895980835, -0.8113636374473572, -0.7625565528869629, -0.03711502254009247, -0.05516029894351959, 0.12016991525888443, -0.08031770586967468, 0.4670046865940094, 0.4142645299434662, 0.0741935446858406, -2.1927103996276855, -2.511403799057007, -0.2063339203596115, -0.9502100348472595, 0.1632484495639801, 1.5788846015930176, -0.041944608092308044, -0.6157426238059998, -1.0365597009658813, -0.09438003599643707, -1.0563340187072754, -0.02368614450097084, -1.2706184387207031, 3.1110682487487793, 0.057391371577978134, 0.5298255681991577, 1.8122332096099854, -0.11638385057449341, -0.46727454662323, -1.232142686843872, -1.2298855781555176, 0.07871347665786743, 0.3623833954334259, 0.16769669950008392, -0.017656905576586723, -0.25286293029785156, -1.218394160270691, 1.614192008972168, 0.5479651093482971, 0.7699491381645203, 0.008340712636709213, 0.06981159746646881, 0.7141218185424805, 0.06730969995260239, 0.15707524120807648, -0.11204014718532562, -3.2596499919891357, 0.6158109903335571, -0.22466006875038147, -0.1357184201478958, -1.111301064491272, -1.8329640626907349, 0.7763025164604187, 0.14053235948085785, 0.032718464732170105, 0.22202295064926147, 0.1584552526473999, 0.026705529540777206, 0.6306086182594299, -1.4768816232681274, -0.7456932067871094, 0.01690366119146347, 0.8636085391044617, 2.1615378856658936, -0.006182210054248571, 1.2937748432159424, -0.1986714005470276, 0.025079555809497833, 0.09521469473838806, 0.2265998274087906, 0.5015494227409363, 0.4117738902568817, 0.007175056729465723, 0.06880563497543335, 0.04157257452607155, -0.05942799896001816, -1.5090235471725464, 0.3082001805305481, 0.30494391918182373, -0.006137371528893709, -0.6880788207054138, 3.3229706287384033, 0.01987527683377266, 0.5697806477546692, -0.09780903160572052, -1.4007657766342163, 0.7860233783721924, 0.23977543413639069, 0.3167031705379486, -0.33438149094581604, 0.7776986956596375, -0.49033933877944946, -0.08397426456212997, 0.02365744486451149, 1.0702449083328247, 0.009304175153374672, -0.012336289510130882, 0.2962930202484131, 0.4273886978626251, 0.6584330201148987, -0.5444769263267517, 0.14943082630634308, -0.006058645900338888, -0.426940381526947, -0.5316919684410095, -0.0010585644049569964, 0.3522475063800812, -0.5794159173965454, 1.1113296747207642, 0.6389779448509216, -0.07767536491155624, -0.19067727029323578, 0.9895714521408081, -0.05728449672460556, 0.4519238770008087, 2.514592170715332, -0.9391498565673828, 0.06564565002918243, -1.2681628465652466, -0.9272416234016418, -1.9590208530426025, -0.01797725073993206, -0.389180451631546, -0.663359522819519, -0.30746740102767944, -0.0003761289408430457, -1.3687275648117065, 0.021217375993728638, -0.3685658276081085, -0.21276623010635376, 0.025993455201387405, -0.03155452758073807, -0.15380582213401794, -0.023700134828686714, -2.4768621921539307, 1.780379056930542, -1.0527867078781128, 0.1673276424407959, 2.331434488296509, 2.478226900100708, -0.026069382205605507, 0.1760614514350891, 0.03651025891304016, 0.7100759744644165, 0.2142792046070099, -0.6217435598373413, -1.242195725440979, -0.11716780811548233, -2.350449323654175, -0.6349963545799255, 0.031612444669008255, -1.4553921222686768, 0.28407612442970276, 3.326960563659668, 0.38905519247055054, 1.0088400840759277, -0.023090794682502747, 0.8159393668174744, 0.6265515685081482, -0.09428343921899796, 0.5737462043762207, 1.017669677734375, 0.0015916345873847604, -0.8607180118560791, -0.18248382210731506, 1.0022629499435425, 0.3784950077533722, 0.10298794507980347, -0.14683330059051514, 0.0790053978562355, -1.2826967239379883, -0.07563747465610504, 0.06397431343793869, -1.403084397315979, 0.847943127155304, -0.5277644991874695, 0.33921128511428833, -0.1430465430021286, -0.25760456919670105, -0.40921664237976074, -0.3381445109844208, 0.2947077751159668, 0.2489100992679596, 0.09472320973873138, 0.8614010810852051, -0.500147819519043, 0.14743611216545105, 0.5890995860099792, 0.6704449653625488, 1.9036136865615845, -0.49450400471687317, -0.2640567719936371, 0.32990652322769165, -0.07129751890897751, -0.3373906910419464, -1.0617061853408813, -0.461723655462265, 1.0166703462600708, -0.27649667859077454, -2.5105373859405518, 0.5826129913330078, 0.8764545917510986, -0.014123829081654549, 0.3742976188659668, -0.47933429479599, 0.7079532742500305, 1.8282488584518433, -0.375750333070755, 2.5297062397003174, -0.170492023229599, -0.5145624876022339, -0.48936590552330017, 0.050896838307380676, 0.9801042675971985, -0.39874914288520813, 0.19391977787017822, 0.08946575224399567, 0.6689327359199524, 0.4812246859073639, 0.8811618089675903, 0.6638361215591431, 1.0726641416549683, -0.20282040536403656, -1.1097761392593384, 0.07080470770597458, 0.0006010659271851182, -0.2170851230621338, -0.11693922430276871, -0.9148389101028442, 0.47516319155693054, -0.6321216821670532, 0.06176263466477394, 0.5011616349220276, 5.863566875457764, 1.889953374862671, -1.394810676574707, -0.24580205976963043, -0.5327339768409729, 0.26740002632141113, 0.058408599346876144, -0.7783113718032837, 0.14241361618041992, 0.06077045202255249, 0.14346973598003387, -0.24164283275604248, -0.9572628736495972, -0.02928338758647442, 0.5202733874320984, 0.5063598155975342, 0.39135411381721497, 4.219406604766846, 1.1019823551177979, 0.4149470031261444, 0.13073104619979858, 1.0968420505523682, 0.5595387816429138, -1.75374174118042, 0.17865979671478271, -0.4236419200897217, 0.40106001496315, -0.4557023346424103, 1.3181805610656738, 0.01417784858494997, 0.01417999342083931, -0.016195593401789665, 0.8693896532058716, 0.25599023699760437, 0.4867822229862213, 0.04094976931810379, -0.12074488401412964, -0.12219925969839096, -0.41487517952919006, 0.1522078961133957, 0.3726680874824524, 0.024349937215447426, 1.4487298727035522, -0.35023728013038635, -0.05731665715575218, 0.054665397852659225, 0.0160883367061615, 0.8740981817245483, -1.0108253955841064, -0.614703357219696, 0.06310775876045227, -0.09131144732236862, -0.12719370424747467, 1.237718105316162, -0.04251226782798767, 0.16264532506465912, -0.030255161225795746, -0.029130486771464348, -0.0005814759060740471, -0.18577250838279724, 0.22588075697422028, -0.09651928395032883, -0.12006782740354538, -0.15476949512958527, 0.43874675035476685, 1.5336607694625854, -0.8384946584701538, 1.3713080883026123, 1.045563817024231, 0.493197500705719, 0.11409781128168106, 1.5832194089889526, -1.755749225616455, 0.005080910865217447, -0.42275670170783997, 0.2107914686203003, 1.1466429233551025, 0.40116074681282043, 0.5572361350059509, 0.49302420020103455, 0.6359885931015015, 0.4137597680091858, -0.8048712015151978, 2.079118490219116, -1.7360862493515015, 0.008505448698997498, -0.057153310626745224, 0.017665529623627663, 0.47170525789260864, -0.790187418460846, 0.09221061319112778, -0.09004142135381699, 0.15512670576572418, 0.5342792272567749, 0.9868701696395874, -0.01864670217037201, 0.2510550022125244, -0.025625640526413918, -1.5184240341186523, -0.535669207572937, -0.15896207094192505, -0.3059563934803009, -0.2138313502073288, -0.896960437297821, -0.5605534911155701, -0.17875626683235168, 1.2012896537780762, 1.1499367952346802, -0.0028350576758384705, 1.3752093315124512, -0.7897595763206482, 0.004176404792815447, -0.21005448698997498, -0.8423174023628235, 0.13810694217681885, 0.0059208651073277, 0.5377572774887085, 3.572699546813965, 0.0003921546449419111, -0.36729544401168823, 0.3668799102306366, 0.03154231607913971, -0.000881112297065556, -0.30826491117477417, 0.7741199731826782, -3.162365674972534, -0.1437293440103531, 1.3152670860290527, 0.40547773241996765, -0.028542522341012955, 0.0071463086642324924, 0.29611265659332275, -1.0835366249084473, -0.686323344707489, 0.33773621916770935, -0.5409526228904724, 0.15638452768325806, -0.6260493993759155, -0.008784999139606953, 0.1965986043214798, -0.3888106346130371, 0.5209544897079468, 0.2778438925743103, 0.8800210356712341, -0.023189345374703407, -0.2106366604566574, -0.12539757788181305, -2.2033910751342773, 0.00038707375642843544, 0.2662275433540344, 0.9050017595291138, -0.7257285714149475, 0.8759747743606567, 0.6613463759422302, -0.05433519929647446, -0.8895323872566223, -0.17022167146205902, -0.45822930335998535, -1.3630980253219604, -2.6835954189300537, -1.5665665864944458, -0.40805357694625854, -0.006174131762236357, -0.5515613555908203, -0.11760357022285461, 0.7478171586990356, -0.07544458657503128, 0.3274097442626953, 0.6850005984306335, 3.5789339542388916, -0.02115696668624878, -0.26987969875335693, -2.4527482986450195, -0.3395160138607025, 1.5733155012130737, 4.699654579162598, -0.5619794726371765, 2.1710450649261475, -0.7287208437919617, -0.09535036981105804, -0.10505657643079758, -0.34200212359428406, 1.0239324569702148, -0.5535993576049805, -0.1769130825996399, 1.5157127380371094, 0.18975012004375458, 0.31260156631469727, -2.0346813201904297, 0.06401955336332321, -0.26020753383636475, 0.16116392612457275, -0.175297349691391, 1.410258173942566, 0.009500640444457531, 0.24204842746257782, -0.05952559411525726, 0.4294116795063019, -0.37860625982284546, -1.3752858638763428, 0.0019158551003783941, 0.05477217584848404, -0.010716276243329048, -0.2686537504196167, 0.0034732904750853777, -0.33145371079444885, 0.3441353738307953, -0.024242762476205826, 0.05628166347742081, 0.0508098341524601, -0.06346028298139572, -0.7407578825950623, -0.3283921480178833, -0.22157920897006989, -0.21535925567150116, 0.4450680911540985, 1.2046133279800415, 0.06279416382312775, 0.31182774901390076, -0.6919612288475037, 1.0059598684310913, 0.09786651283502579, -1.4462262392044067, 0.4034256935119629, 2.783196210861206, -0.2944386899471283, 1.3955885171890259, -0.5958971381187439, 0.5619673728942871, 0.341947466135025, 0.2636168897151947, -0.5196537971496582, 0.06923743337392807, -0.40012505650520325, -0.24441416561603546, -2.4000394344329834, 0.7634166479110718, 1.5381383895874023, -2.2796478271484375, -0.07176591455936432, 0.42695358395576477, -0.09473664313554764, -0.16684359312057495, -0.10446923226118088, -2.1302802562713623, 0.011420628987252712, 0.49201586842536926, -0.046550679951906204, 0.11995846778154373, 1.6055598258972168, 0.17969156801700592, -0.013127731159329414, 0.05283304676413536, -0.018782610073685646, -0.00011995471140835434, 0.6075055003166199, -0.5353161096572876, -0.5335531830787659, 0.5337735414505005, 0.4226202666759491, 0.12152041494846344, 0.03596383333206177, -2.354252815246582, 0.2610575258731842, 1.5063387155532837, 0.0366000197827816, 0.2518061101436615, -0.7086812257766724, -0.7852892279624939, -0.49450361728668213, 0.35113707184791565, 1.0178884267807007, -1.3657282590866089, 0.09996014833450317, -1.1300026178359985, -0.005601249169558287, -0.08151093870401382, -0.8965408205986023, -4.033938884735107, -0.0029407632537186146, 0.059321604669094086, -0.03592276573181152, 1.6785985231399536, 0.12127324938774109, 0.6747549176216125, 1.091063141822815, 0.029218364506959915, -0.16128672659397125, 0.3098100423812866, -0.3528992235660553, -0.3078244626522064, -0.002175192814320326, -0.26886308193206787, 1.786902904510498, 0.3162945508956909, 0.3931778073310852, 0.21665368974208832, 0.008774125948548317, -1.0255835056304932, -0.20245245099067688, -0.13114139437675476, 0.7031175494194031, 0.5501707792282104, -0.0048678223975002766, 3.407642126083374, -0.25926122069358826, -0.010570196434855461, -1.0039478540420532, 0.00022747389448340982, -2.1639082431793213, -1.17172372341156, -0.23607051372528076, 3.576502799987793, -0.14406836032867432, -0.13061390817165375, 0.04924945905804634, -0.7083601951599121, 0.016710352152585983, -0.6789001822471619, -0.0049696993082761765, -0.7091825008392334, 0.19793429970741272, 0.04603513702750206, 0.08633538335561752, 0.04362015426158905, -2.767622470855713, 0.7470807433128357, -1.2182769775390625, -0.08852784335613251, 0.6865098476409912, 0.15914452075958252, 0.8526677489280701, -1.1486726999282837, 1.4158045053482056, -1.8144131898880005, 0.04864244908094406, 0.01859852857887745, 1.4608160257339478, 0.002773027867078781, -0.0714370384812355, 0.18255110085010529, 0.20979954302310944, 0.05180499330163002, 0.012460583820939064, 0.2048458755016327, 0.06885357201099396, 0.6545825600624084, 0.8282763957977295, -0.3969103693962097, 1.580811858177185, 0.05095143988728523, -0.6675646305084229, 0.060501206666231155, 0.8076764345169067, -0.7017820477485657, 0.024356866255402565, 0.03792278468608856, 0.2048415094614029, 1.514399766921997, 0.018917735666036606, -0.992742121219635, 0.24897408485412598, 0.017887061461806297, 0.31270575523376465, 0.07719887048006058, -0.10153371095657349, 0.16720595955848694, -3.7223453521728516, -1.7931674718856812, 0.15220734477043152, -0.2727908790111542, -0.9817190766334534, 0.4340577721595764, -0.15769930183887482, -0.8428941369056702, 0.23974114656448364, 0.19457577168941498, -1.0381652116775513, 1.0778520107269287, 0.2542765438556671, 0.313114732503891, 1.80911386013031, 1.3353654146194458, -1.0221531391143799, 0.3815785050392151, -0.01916416734457016, -0.006246904376894236, 0.11668649315834045, 0.7109670639038086, 0.5524291396141052, 0.2943197786808014, -0.4291158616542816, -0.10232485085725784, 0.46923837065696716, 0.8402569890022278, 0.10226580500602722, 0.2126234471797943, 0.06226088106632233, 1.1617048978805542, 0.0014214105904102325, 0.03927403315901756, 0.5053480267524719, -0.029038425534963608, 0.7861323356628418, -0.49672383069992065, -0.0010978439822793007, 0.037924874573946, -1.4441827535629272, -0.19091343879699707, -0.35954195261001587, -0.8279598355293274, -1.790488600730896, -1.3453015089035034, -0.11031731218099594, 0.34898677468299866, -0.5364489555358887, 0.04122895374894142, -0.5679669976234436, 0.14086875319480896, 0.6902444958686829, 0.10289139300584793, 0.14980754256248474, 0.956786572933197, 0.5887985825538635, -1.0644557476043701, 0.0784008651971817, -0.1052541509270668, 0.04130258411169052, -0.6057370901107788, 0.001772275660187006, 0.2520502209663391, -1.4141936302185059, 0.03846895694732666, 0.0049904282204806805, -0.04131553694605827, -0.2710017263889313, 0.7901182770729065, -0.05817268416285515, -0.35380831360816956, 0.009215064346790314, 1.973183035850525, -0.22540859878063202, -0.22396938502788544, 0.6292229890823364, 0.032793495804071426, -0.8271920084953308, 0.11850292980670929, 1.4096719026565552, -0.3875952959060669, -0.238531231880188, 0.8192628622055054, -0.1690816730260849, 0.5578420758247375, 0.001171368290670216, -0.1741093248128891, 0.027478206902742386, 0.26948556303977966, 0.18375925719738007, -0.10477600991725922, 1.1396092176437378, 0.0657329335808754, -0.1058545634150505, 0.16146059334278107, 2.9622154235839844, -0.9494338631629944, -0.10211740434169769, -0.626911461353302, -0.38989895582199097, -3.4155609607696533, 0.25140926241874695, 0.03678334876894951, -0.32607242465019226, 0.18988020718097687, -0.181517094373703, -0.05300010368227959, 0.23901401460170746, 0.12484987080097198, 0.45112407207489014, 0.055705416947603226, -0.16285845637321472, -1.0095564126968384, -0.14908833801746368, 1.205134630203247, 0.0052190194837749004, 0.19640758633613586, 0.010075386613607407, -0.5432479977607727, 0.7885117530822754, -0.7200925946235657, 0.9964758157730103, -0.14843392372131348, -0.9060329794883728, -0.08474884927272797, -3.8026506900787354, 0.1262197345495224, 0.5786678194999695, -0.2727384567260742, -0.8158866167068481, -0.4645295739173889, 0.10601314157247543, -0.16810081899166107, -1.148573875427246, 0.5763727426528931, -0.4877737760543823, 0.5466518402099609, -0.9424012303352356, -0.6953147649765015, -0.022048931568861008, -0.3792881965637207, -0.0761520266532898, 0.5968884825706482, -0.3804475665092468, 0.03351673483848572, -0.08141006529331207, -1.364127516746521, 0.07243713736534119, -0.40260303020477295, 0.002278289757668972, -1.8994101285934448, 0.9975311756134033, 0.6399073600769043, 1.7337456941604614, -0.014436799101531506, 0.27799683809280396, 0.1390027403831482, 0.17229269444942474, -0.08716867119073868, 0.43212011456489563, -0.6482419967651367, 0.7815166711807251, -0.28610292077064514, -0.24053700268268585, 0.006097432225942612, -0.2585941553115845, -1.5740699768066406, -0.011296744458377361, 0.8463615775108337, -0.06913338601589203, -0.297967791557312, -0.5124375820159912, 0.20621168613433838, 0.7923486828804016, 0.2474350482225418, -0.6634584665298462, 0.19493289291858673, -0.039463676512241364, -0.32450634241104126, 0.10713542997837067, -0.38488104939460754, 0.10990286618471146, 0.05744626000523567, -0.02837028168141842, 0.1784348040819168, 0.5691158771514893, 0.31102222204208374, 1.7545819282531738, -0.7068235278129578, -0.39627763628959656, 1.3881478309631348, 0.6452415585517883, 0.15887175500392914, -1.5500739812850952, -0.06670559197664261, 0.8252142071723938, -0.4868375062942505, -0.32960245013237, 0.7252506017684937, 0.9762352108955383, 0.08885884284973145, 0.2589382827281952, -0.9817754626274109, -1.4571962356567383, -0.0032660479191690683, 0.4804574251174927, -0.424545019865036, -0.858948826789856, 1.0496869087219238, -0.03961114212870598, -0.45000579953193665, 0.010448463261127472, -1.8617948293685913, -0.4507942497730255, -0.4946261942386627, 2.9528589248657227, 0.1121254488825798, 0.26496386528015137, 0.24186722934246063, 0.3845483958721161, 0.17542612552642822, -0.013830865733325481, 0.5167355537414551, 0.16314350068569183, 0.15781643986701965, 0.14750270545482635, -0.8354567885398865, -0.3432203233242035, 1.3258605003356934, 2.2638518810272217, 0.8986931443214417, -0.1735927015542984, 1.1952250003814697, -0.5376575589179993, 0.21345441043376923, -0.3616369068622589, -0.02460504323244095, -0.007216781377792358, 0.5401068925857544, 0.6832994818687439, 2.008997917175293, 0.11442318558692932, 0.8552919030189514, -0.42323023080825806, -0.5771229267120361, 0.7520872950553894, -0.80534428358078, 0.2106066346168518, 3.2132649421691895, -1.195852279663086, 0.27931880950927734, 1.9854941368103027, 1.6337536573410034, -0.994574785232544, 1.2665091753005981, -0.3128420114517212, 1.3199986219406128, 0.3555726706981659, -0.00647016754373908, -0.16889454424381256, -1.8988230228424072, 2.3497886657714844, 1.4812978506088257, -0.5697486996650696, -0.02332044206559658, 1.4913311004638672, -0.03691241145133972, -0.13378162682056427, -0.46006885170936584, 2.7422678470611572, 0.3852234482765198, -0.10329616814851761, -0.03532105311751366, -0.011861713603138924, 0.01345937978476286, -0.017123064026236534, -0.6861565709114075, 2.046067953109741, 0.038752056658267975, 0.04461343586444855, -1.6974936723709106, 0.6712268590927124, -1.9423630237579346, -0.029404444620013237, 1.241635799407959, 0.03396112844347954, -0.10078319907188416, 3.0245168209075928, 1.5426219701766968, 1.1844837665557861, -0.16433201730251312, -1.5262682437896729, -1.3789056539535522, 0.4144515097141266, 0.7417467832565308, 0.4731617271900177, 1.3851892948150635, 0.0015365590807050467, -4.229673862457275, 0.06152130290865898, 0.46913546323776245, -0.03453902155160904, -0.26665255427360535, 1.290510654449463, -3.3782458305358887, -0.00021641861530952156, -1.3029621839523315, 0.11992684751749039, -0.20957084000110626, -1.5634047985076904, -0.19457373023033142, 2.759169816970825, 2.831068515777588, -0.9691383838653564, -0.0027457033284008503, -0.09642443805932999, 0.06977646052837372, 1.4907441139221191, -0.4103904068470001, -0.10125751793384552, -0.2455800175666809, 0.34439554810523987, 0.18286550045013428, 0.05158798396587372, 0.456866979598999, -0.8151218295097351, -1.7448331117630005, 0.48851314187049866, 0.20769049227237701, 1.2362457513809204, 0.020486291497945786, -0.31725627183914185, -0.16236843168735504, -0.1521565318107605, -1.6109944581985474, -0.1705494374036789, 0.5675185918807983, -0.16469928622245789, 0.13124790787696838, -0.30046072602272034, -1.0676729679107666, -0.03842209652066231, 1.125907063484192, 0.941107451915741, -0.2985447347164154, -0.026121698319911957, -2.7033233642578125, 0.7355188131332397, 1.8730899095535278, -0.0359678789973259, -1.6195164918899536, -1.9939886331558228, 0.03417046740651131, 0.6371975541114807, 0.08172854036092758, -1.182707667350769, -0.3402250409126282, 0.10539009422063828, -0.15043458342552185, 0.521521270275116, -0.048448849469423294, -0.34065771102905273, -0.24784459173679352, -1.4491645097732544, 0.24609920382499695, -0.07893011718988419, -0.016366878524422646, 2.316586971282959, 0.989338219165802, 0.48336467146873474, 0.25085628032684326, 2.0127670764923096, 0.4489557445049286, 0.2430315762758255, 0.10438651591539383, 0.850435197353363, 0.15263931453227997, 1.5374575853347778, 0.08563809841871262, -0.4313672184944153, -0.040262266993522644, -0.9224111437797546, -0.15844902396202087, -0.2744525671005249, -0.16081053018569946, -0.0054935612715780735, 0.010343153029680252, -0.9218664169311523, 1.866543173789978, -0.030477937310934067, 0.2654840350151062, -0.2081352025270462, -0.2999463975429535, -0.4818660318851471, -0.13114804029464722, 0.5319490432739258, -1.8729469776153564, 1.1573089361190796, 0.1483534276485443, -1.4576796293258667, 0.5280497074127197, -0.5802725553512573, -0.02948054112493992, 0.05394887179136276, 1.3933217525482178, 0.050621382892131805, -0.3339630961418152, 0.016722917556762695, 1.5220375061035156, 0.40929391980171204, -0.6066373586654663, -0.08976570516824722, -0.07152215391397476, 1.0509942770004272, -0.13133418560028076, -0.32794493436813354, 0.14595529437065125, -0.46937891840934753, 0.0875142365694046, 1.4242846965789795, 0.7707566618919373, 0.044839855283498764, -0.04536869004368782, -0.5399628281593323, 0.15651164948940277, -0.1538083702325821, 0.05693701282143593, 0.029690448194742203, -0.3026127517223358, 0.17394278943538666, -1.2033337354660034, -0.008347876369953156, -1.3140435218811035, -0.005631304811686277, -0.7570619583129883, -0.8904990553855896, -0.3730641007423401, 0.027471061795949936, -0.011860411614179611, -0.6061801910400391, 0.6874816417694092, 1.083216905593872, 0.2526790499687195, -0.7444344758987427, 0.04860721156001091, 0.80857914686203, -0.162677600979805, 3.887708902359009, 0.14702291786670685, -0.9557831287384033, 0.6013851761817932, 0.3710058033466339, 2.061302423477173, 2.84773850440979, -0.03356855362653732, 0.16956250369548798, 0.26544708013534546, 0.16857825219631195, -0.6501940488815308, -0.07686235755681992, -0.07137694209814072, -0.04482785239815712, -0.0032336986623704433, 0.07029150426387787, 0.07644451409578323, 0.07744385302066803, 0.17255213856697083, 0.883059561252594, -0.42940354347229004, -0.3193054497241974, 0.37764090299606323, -0.7428762912750244, -0.08603421598672867, 1.1106125116348267, 0.43905654549598694, -0.008824842050671577, -1.115777850151062, 0.4804200232028961, 1.9750823974609375, 0.006622989196330309, 0.12856562435626984, -0.3647828698158264, 0.09965400397777557, -0.5334678888320923, -0.006554506253451109, -0.7351270318031311, 0.22725370526313782, 0.3865065276622772, 2.835759162902832, 0.9671652913093567, -0.13333585858345032, -0.44692665338516235, -1.5466762781143188, 3.2960407733917236, -0.25612539052963257, -0.02204728126525879, 0.02607002481818199, 0.303544819355011, 0.07445846498012543, 0.061358168721199036, -0.20588676631450653, 0.006447553168982267, 0.334085077047348, 0.6497451066970825, -0.8744326233863831, 0.10933003574609756, -0.09521966427564621, -0.24012525379657745, 0.3478061258792877, -0.2961725890636444, -0.08515696972608566, -0.059509143233299255, 0.04677422344684601, 1.8890039920806885, 0.3557165861129761, 0.08605386316776276, 1.2439063787460327, -0.6042230129241943, 0.219742551445961, -0.5714155435562134, 0.8227290511131287, 0.09897832572460175, 0.6635156869888306, -0.3710290193557739, -2.721626043319702, 0.12371199578046799, 0.5270915031433105, 1.5878561735153198, 0.043557193130254745, 0.05272386968135834, 0.01503811776638031, 0.64516681432724, 0.8489586114883423, -0.12128390371799469, 0.10750560462474823, 0.33923184871673584, -0.03571765124797821, 0.2588375508785248, 0.2893841564655304, -1.445153832435608, 0.05653781816363335, 0.8788504004478455, -0.17042818665504456, -0.25047072768211365, -3.144965648651123, -1.666782021522522, -0.5015082359313965, 0.21548111736774445, 0.15144182741641998, -1.2942949533462524, -0.27109527587890625, -0.04326024278998375, 0.2079973965883255, -0.6964784860610962, 2.1914901733398438, 0.36862635612487793, 0.17337417602539062, -0.35881146788597107, 0.06274306029081345, 0.26914528012275696, 0.03839677944779396, 0.10057512670755386, -0.44746819138526917, 0.0016367863863706589, -2.4544289112091064, -0.13358190655708313, -0.6231911182403564, -0.5306037068367004, -0.00022026599617674947, -2.1757631301879883, -0.29440072178840637, -2.1861681938171387, -0.3772633671760559, 0.7977197766304016, -0.015123486518859863, 0.007065870333462954, -0.13024602830410004, -0.07075858861207962, 0.08891145139932632, -0.14380063116550446, 1.2109434604644775, 0.012289020232856274, 0.07803946733474731, -1.5050824880599976, -1.01247239112854, -0.019801579415798187, 0.8903457522392273, -0.25349748134613037, -0.026618478819727898, 1.282350778579712, -0.6501035690307617, 0.3697408437728882, -0.22266942262649536, 0.24879693984985352, -0.6100223660469055, 0.04672987014055252, 0.3643050491809845, 0.9963428378105164, 0.01475931890308857, -0.35046887397766113, 0.280679315328598, -0.30197030305862427, -0.023280762135982513, -0.14912253618240356, -7.808931827545166, 0.020338011905550957, -0.011686322279274464, -0.17976519465446472, 0.0514715313911438, 1.1150188446044922, 1.9326012134552002, -1.3768640756607056, -0.14212024211883545, 0.06502202153205872, -1.063143014907837, 0.752069354057312, 0.7018921375274658, 0.022843947634100914, -1.6775298118591309, -0.09512365609407425, -0.5296079516410828, 0.084662064909935, 0.45591193437576294, 0.08271515369415283, 0.20119579136371613, 1.522560715675354, 0.2677527666091919, 0.002242452697828412, 0.4916837513446808, -0.32226675748825073, 0.3297435939311981, -0.0056313928216695786, 0.5878013968467712, -0.8777813911437988, 0.495589941740036, -0.12402443587779999, -0.6735374331474304, 0.08820250630378723, -0.3478066325187683, -0.316101998090744, 0.11592644453048706, -0.3420499861240387, -0.26563355326652527, 0.8652119040489197, 0.19315101206302643, 0.0004624469147529453, 0.3352660536766052, -2.9340786933898926, 0.4665777087211609, 1.097622275352478, -0.14517895877361298, 0.27516984939575195, -0.43711593747138977, 0.05108793452382088, -1.344624400138855, 1.6534844636917114, -1.1356806755065918, -0.4033432900905609, 0.12032733857631683, -0.17934103310108185, -0.017391644418239594, -0.4488234519958496, 0.14510087668895721, 1.5088344812393188, -0.003270896850153804, -0.7500905990600586, 3.5210065841674805, 0.11419344693422318, -0.6258062720298767, 0.42109546065330505, 0.6302947402000427, -0.04515454173088074, -1.5920696258544922, -0.022831551730632782, 0.007168406154960394, -1.3331313133239746, -1.6293786764144897, 1.2417473793029785, 0.7801384925842285, -0.052177201956510544, 0.2787249982357025, 1.498343586921692, 0.7912448644638062, 0.17016927897930145, 0.20423825085163116, 0.6119561195373535, 0.982567548751831, -0.019401200115680695, -0.013199534267187119, -0.1767294555902481, -0.27873679995536804, -0.06617232412099838, 0.177683487534523, 0.8458811044692993, 0.565507709980011, -0.11456679552793503, -2.637072801589966, -0.36525604128837585, -0.15104888379573822, 1.4333752393722534, -0.0105051938444376, -1.3757883310317993, 0.16652409732341766, -0.06699416786432266, -0.11519793421030045, -0.12840589880943298, -0.8463369607925415, -0.08287826925516129, 0.03598557412624359, 1.2816041707992554, -0.14354445040225983, -0.18936112523078918, 3.1322832107543945, 0.03430834785103798, 1.4316399097442627, -1.240858554840088, 0.840424656867981, 0.3332737684249878, -0.19055144488811493, -2.2517974376678467, -0.0003689367731567472, 1.485690951347351, -0.38760900497436523, 0.17637315392494202, -0.8791204690933228, -0.6001194715499878, -0.41277068853378296, 0.6462578177452087, -0.15333165228366852, -0.2801254689693451, 0.7064611911773682, -0.12582507729530334, 0.14106787741184235, -0.4316752552986145, -1.5560894012451172, 0.20739705860614777, -2.9260637760162354, -0.10528949648141861, -0.7086886167526245, -3.255782127380371, -1.5411955118179321, 0.8382432460784912, -0.15369725227355957, 1.2285171747207642, 0.311286985874176, -3.184164047241211, -0.11190231144428253, 1.5652406215667725, -0.7752001285552979, 2.5369443893432617, 0.024920504540205002, -0.04069007188081741, 0.1438976377248764, -0.963291347026825, 0.5185908675193787, -0.0029703830368816853, -0.057336777448654175, 0.7540187239646912, -1.7243908643722534, -2.56083083152771, 0.14137430489063263, 0.7994971871376038, 0.5016170740127563, 0.026016702875494957, -0.4729771316051483, -0.0973338782787323, 1.070415735244751, 0.20640535652637482, 0.5954042077064514, 0.44432201981544495, -0.4568560719490051, 0.1768132746219635, 0.7695648670196533, -0.05344284698367119, -0.05710437148809433, -0.32971620559692383, 0.3677173852920532, 0.9822878241539001, -0.05711188167333603, -0.6915295124053955, -0.023406609892845154, -0.01857353001832962, -1.8589179515838623, 0.07973125576972961, -0.8742988109588623, -0.29611673951148987, 0.1485929936170578, 0.25971993803977966, -0.08280236274003983, -1.2173912525177002, -1.615037202835083, 0.8991007804870605, -0.0073033845983445644, -0.19725944101810455, -1.5657371282577515, -0.12085272371768951, 0.21505920588970184, -1.5575289726257324, 3.729496955871582, 0.4091419577598572, -0.7257548570632935, 3.84732723236084, -0.8328635096549988, 0.053779661655426025, -1.5844882726669312, 0.7325783967971802, -0.2909080684185028, -0.6650430560112, -0.4986930191516876, -0.0752619057893753, -0.0767633318901062, -0.2868674695491791, 1.125699758529663, -0.04135771095752716, 0.5042868256568909, 0.8828839063644409, -0.5547734498977661, 0.006797926500439644, -1.5062196254730225, 0.2162739336490631, 0.013208740390837193, 2.4750969409942627, -0.7865333557128906, 0.22595568001270294, -0.21445618569850922, -0.37819117307662964, -0.05199110135436058, -0.2280561625957489, 1.1538667678833008, -0.2052740901708603, -0.5369973182678223, -1.9717423915863037, 0.9927289485931396, 0.9065373539924622, 0.7357764840126038, -1.0306130647659302, -4.414975643157959, -0.3361840844154358, 0.4788433611392975, -0.46727436780929565, -1.0375465154647827, -0.031580954790115356, 0.2765031158924103, 0.977572500705719, 1.5996322631835938, -0.14222891628742218, 0.0022019334137439728, -2.054816722869873, -0.009195253252983093, 0.0406259186565876, -0.7469806671142578, 0.20121946930885315, 0.12488943338394165, 0.14848588407039642, -0.1863786280155182, 1.0144211053848267, 0.42150282859802246, -0.6254614591598511, 0.12733007967472076, 0.2279203087091446, 0.07279382646083832, 1.3394544124603271, -0.23286955058574677, -0.047793589532375336, 1.228657841682434, -0.111839160323143, 1.8455041646957397, 0.05662829056382179, 0.07229822129011154, -2.091322422027588, 0.5217210650444031, -0.920587956905365, -0.7509424686431885, 0.21189548075199127, -0.6180030107498169, -1.8636703491210938, 0.01657126657664776, 3.293570041656494, 0.08897590637207031, 0.07536699622869492, -0.21918871998786926, 0.014121298678219318, -0.9296945333480835, 0.476622998714447, 0.2956855893135071, 1.0335941314697266, -0.056748759001493454, -0.9073507189750671, 1.2405261993408203, 0.3023809790611267, 0.6602077484130859, 0.05280589312314987, 1.2359410524368286, -2.4561221599578857, -0.09745894372463226, -0.768193781375885, 0.34716784954071045, -2.964719533920288, -1.0639677047729492, -1.7333283424377441, 0.0006021708832122386, 0.04967166483402252, 0.28803181648254395, 0.18883052468299866, 0.230041965842247, 0.6757861375808716, -0.956271231174469, 0.39010322093963623, -1.6922180652618408, -0.33977842330932617, 0.014350577257573605, -0.3359983265399933, -0.0922011211514473, -0.18632115423679352, -0.6225718259811401, 0.861511766910553, 0.021188722923398018, 1.4664250612258911, 1.528245210647583, -3.0423200130462646, -0.891123354434967, 0.31862813234329224, -0.2966594696044922, -1.37062668800354, 0.17775805294513702, -1.6436172723770142, -0.9756033420562744, 1.0801769495010376, -0.4277006983757019, -1.1743576526641846, 0.10553094744682312, 0.24255476891994476, 4.159043312072754, 0.4100491404533386, 0.3857841491699219, 1.0073554515838623, 0.39728084206581116, 0.09924538433551788, 0.722139835357666, -0.34999290108680725, -0.42394569516181946, 1.2744762897491455, -0.20268218219280243, -0.9166375994682312, 0.3206184208393097, -0.1965249925851822, 1.4058009386062622, 0.17287422716617584, 0.027742788195610046, 0.07073215395212173, -0.03433579206466675, -0.09067418426275253, -0.43960630893707275, 0.011151297017931938, 0.2321842461824417, -0.02167416922748089, -0.21588926017284393, -0.24606510996818542, 1.2794551849365234, -1.698883295059204, -1.1683748960494995, 0.3033422827720642, 0.591255247592926, -0.28247272968292236, 0.20891797542572021, -0.22140388190746307, -1.5287882089614868, -1.1768463850021362, 0.5208702087402344, -0.11549592018127441, 0.921862781047821, 0.12596729397773743, -2.8852200508117676, 2.4051008224487305, 0.06438140571117401, 0.1489650309085846, -0.04926907643675804, -0.3254977762699127, -1.5033074617385864, 0.015720833092927933, -0.038667984306812286, -0.35150066018104553, -0.40199363231658936, 0.7434049844741821, 0.10497744381427765, -1.2116165161132812, -1.1758582592010498, 0.31466221809387207, -0.19468669593334198, 0.35466355085372925, -0.910890519618988, 0.13670387864112854, -0.023175710812211037, -0.9306608438491821, 0.15294288098812103, -1.518956184387207, 0.7315111756324768, -0.10914872586727142, 0.4836151897907257, 1.4095985889434814, 0.3985741138458252, -1.1018824577331543, 0.9135065674781799, 0.9833288788795471, -0.6426811814308167, 0.38679423928260803, -0.2606075704097748, -0.5694538950920105, -0.2916001081466675, -0.31542065739631653, 0.505612850189209, -1.382237434387207, 0.03539885953068733, 0.340973436832428, 0.2646143436431885, -0.028105199337005615, 0.6594889760017395, 0.35358020663261414, -0.2331031709909439, 0.016160300001502037, 0.19854432344436646, 1.0004098415374756, -0.24407000839710236, 0.08594673126935959, 0.18310096859931946, -0.09066365659236908, -0.4710742235183716, 0.5841597318649292, -0.7856971025466919, 0.11602127552032471, -0.7814630270004272, 0.00823400542140007, 1.1041935682296753, -0.6223012804985046, 0.46985283493995667, -1.0741889476776123, -0.19862237572669983, 3.1847360134124756, -1.4763529300689697, 0.602344810962677, 0.032033130526542664, 0.253793865442276, 0.05703825131058693, -0.5389286279678345, 0.1759357750415802, 0.09337697923183441, 0.05016804113984108, -0.07767266035079956, -1.304619312286377, 0.9764994382858276, 0.2659878432750702, 0.017815520986914635, 0.46815502643585205, -0.24442847073078156, 0.13424593210220337, 0.1407558172941208, 1.4659980535507202, -0.1331557184457779, -0.085108682513237, -0.013105533085763454, 0.44160205125808716, 0.11709899455308914, -0.018644575029611588, 1.6839039325714111, -1.7785954475402832, 0.2867431640625, -0.6915178298950195, 0.34995248913764954, -1.5076359510421753, 0.06204310804605484, -1.3082131147384644, 0.8894218802452087, -0.08075512945652008, -2.5501291751861572, -0.3097222149372101, 0.010649862699210644, 0.7210038304328918, 0.16840602457523346, -0.1359788179397583, 0.5884751677513123, 1.5756540298461914, -0.439199298620224, 0.9110971093177795, -0.08567317575216293, 0.3223302662372589, -0.47888392210006714, 0.10872513055801392, 0.057706303894519806, -0.756294310092926, 0.2756596803665161, -0.32683026790618896, 0.4253283441066742, 0.0002165302139474079, -0.08181571215391159, 1.6762408018112183, 1.4115004539489746, 0.28163519501686096, -0.2481779009103775, 0.08111903816461563, 0.0342019759118557, 0.22413939237594604, -0.9554696679115295, -1.5146088600158691, -0.308722585439682, -0.49827682971954346, -0.015923574566841125, -0.010030184872448444, -0.2312159687280655, 0.16299961507320404, -0.22641989588737488, 0.15978603065013885, -1.1340874433517456, -0.5606329441070557, -0.4494810998439789, 0.039971351623535156, -0.6185551881790161, 0.2813316881656647, 0.9756386876106262, 0.7297704219818115, -1.0781949758529663, 0.7910505533218384, -0.013290846720337868, -1.9054020643234253, 0.4629438519477844, 0.08676327019929886, -1.7965748310089111, 0.9168482422828674, -0.2659641206264496, -0.789825975894928, 2.2006425857543945, 0.041850343346595764, 1.5715583562850952, -0.9548721313476562, 0.21519483625888824, -0.3090956509113312, 0.6173868775367737, -0.00022543237719219178, 0.056839410215616226, -0.2881580889225006, 0.07131035625934601, 0.5168808698654175, -0.9247894883155823, 2.759307622909546, -1.7349364757537842, 0.6033177971839905, 0.15534688532352448, 0.1903284341096878, 0.03381485491991043, -0.9551721215248108, 1.574755311012268, -0.46154820919036865, -0.02865031361579895, 1.79734206199646, 0.8308823704719543, 0.24744336307048798, 0.14603431522846222, -0.03990757837891579, -0.010254713706672192, -0.7123523354530334, -0.13378021121025085, -0.7349859476089478, -0.2521584928035736, -0.10652468353509903, -2.39100980758667, 2.4221105575561523, -0.13155429065227509, -0.17129698395729065, 0.8912932276725769, -0.19764181971549988, -0.19309544563293457, -0.2275221347808838, 0.0010934153106063604, 0.06848373264074326, -0.279798299074173, -4.328932762145996, -1.1749353408813477, 0.3269484043121338, 3.285095691680908, 0.7491506934165955, 1.591691493988037, -1.4870288372039795, 0.1841134876012802, -1.197167992591858, -0.15000158548355103, -0.49487441778182983, 0.7434058785438538, -0.09903998672962189, 0.6119300723075867, -0.40160244703292847, -2.0336692333221436, -0.0855044350028038, -1.123119831085205, -0.6325225830078125, -0.5056108236312866, -0.3232242166996002, 0.5384142994880676, 0.06695900857448578, 0.07787027209997177, 1.0217894315719604, 0.11597985029220581, -0.7102084755897522, 0.3897451162338257, 0.6181854009628296, 0.11942029744386673, 0.6682107448577881, 0.19981178641319275, -0.2865583598613739, -0.02529531717300415, -0.1215270608663559, 0.09144534915685654, -0.816815197467804, -0.02674734964966774, 0.8706364035606384, 1.9018372297286987, 0.05140978842973709, -1.0173791646957397, 0.18255847692489624, -0.5984890460968018, 2.1792349815368652, 0.17559300363063812, 0.39632028341293335, 0.43455690145492554, 0.5945756435394287, 1.944542646408081, 1.017238974571228, -0.03319088742136955, 0.09273185580968857, 0.399729460477829, 0.6317533254623413, -0.7313641309738159, -0.3350205421447754, -0.9452535510063171, -0.5964296460151672, 0.016192300245165825, 3.176780939102173, -0.15505410730838776, 0.5051915645599365, 0.19543227553367615, -0.10810267180204391, -0.018551236018538475, 0.5939801931381226, -0.18454064428806305, 0.019290562719106674, 0.010952189564704895, 1.5185099840164185, 0.31252017617225647, 1.8560121059417725, 0.21869128942489624, -0.6809683442115784, 2.2005627155303955, -0.049951449036598206, 1.0061794519424438, 0.7427183389663696, 0.6728797554969788, 2.3077235221862793, 0.13649211823940277, -0.0068009961396455765, -0.004689202178269625, -2.068906784057617, 0.011670127511024475, 0.35655638575553894, -0.05045468360185623, -1.0836482048034668, -0.26852184534072876, -2.878082752227783, -0.1274450421333313, -0.013465886935591698, -1.0294196605682373, 0.1429092437028885, 0.8221763372421265, 0.10902303457260132, -0.8511053323745728, -0.11770781874656677, -1.0604548454284668, 0.8042908906936646, -0.15451450645923615, -1.8283915519714355, -0.029516087844967842, 0.4633859395980835, -0.2822364270687103, -0.4082646369934082, -0.5884191989898682, 0.394718199968338, 0.5150681138038635, -0.03507750853896141, 0.08077738434076309, -0.5829781889915466, 0.2355491667985916, 0.05955866351723671, -1.5451191663742065, -0.3459196090698242, -0.05414684861898422, 0.7375516295433044, -0.01935083232820034, 0.5711429715156555, -0.21161678433418274, -0.038471873849630356, -0.8890249729156494, -0.05547131597995758, -0.3395868241786957, 0.2066521942615509, 0.6208837032318115, -1.3201500177383423, -0.0018251319415867329, 0.00657608499750495, 0.447313517332077, -0.7506813406944275, -0.18766458332538605, -1.289753794670105, 0.8670057654380798, 0.7335526943206787, 0.007179903797805309, -2.0816073417663574, -0.16809609532356262, -4.533409118652344, 0.4790462255477905, 0.8141738176345825, -0.6303654313087463, -1.8971853256225586, 0.26141294836997986, -0.8008604645729065, 0.0017403598176315427, -0.9910018444061279, 0.04133119434118271, -0.23036031424999237, -0.031132381409406662, -0.5693494081497192, 0.48897185921669006, 2.021641969680786, -1.2739043235778809, -1.2780482769012451, 0.03800559043884277, 0.24762065708637238, 0.12192156910896301, -0.27676063776016235, 0.996374785900116, 1.4736040830612183, 0.6394641995429993, -0.09558982402086258, 0.5443646311759949, 0.43236419558525085, -0.7190722823143005, -1.0021886825561523, -1.1242510080337524, -0.032312214374542236, -0.5911067724227905, 0.36732324957847595, 0.0904560312628746, -0.46102532744407654, 0.9546043872833252, 0.27707648277282715, -0.5864506363868713, -1.8659870624542236, 0.3082030713558197, -2.3063976764678955, -0.04639121890068054, -0.21931231021881104, 0.2800385653972626, -0.2339378297328949, 0.6503841876983643, 4.65071716462262e-05, -0.938877284526825, 1.3915644884109497, 0.3098367750644684, 1.9500559568405151, 0.2748294770717621, -0.1541987508535385, -0.10915356874465942, 0.023034952580928802, -0.5104792714118958, -0.30116555094718933, 0.005074838176369667, 0.8032810091972351, -0.027723262086510658, -1.7101037502288818, -0.7703421711921692, -0.2758634686470032, -3.4138290882110596, -0.5885831117630005, -0.6651738286018372, -0.27139201760292053, -0.23497489094734192, 0.24337831139564514, 1.309156060218811, -1.037471055984497, -2.5987539291381836, 0.4253150224685669, 0.9910027384757996, -5.169590473175049, -0.004219465889036655, 0.002143353456631303, -0.1172778308391571, -0.7105301022529602, -0.5381073355674744, -2.8881447315216064, -0.6395010352134705, -0.24079622328281403, -0.009446488693356514, -2.6754400730133057, -0.20827104151248932, -0.0800037831068039, -0.2730589807033539, -1.4658387899398804, -1.2807998657226562, 0.15708611905574799, -0.04516715183854103, 0.05692422762513161, -0.9236088395118713, 1.406996250152588, 0.29877203702926636, 0.7776851058006287, 0.0047255330719053745, 1.6672556400299072, -0.021670065820217133, 0.8738225698471069, 0.21428726613521576, 2.1573030948638916, 0.00915099959820509, 0.03925156593322754, 0.25585269927978516, -0.047152742743492126, 0.2914283275604248, 0.7501054406166077, 0.05054491013288498, 0.6037750840187073, 0.5759909152984619, 0.6367218494415283, -2.251812219619751, 0.14703360199928284, -2.360853672027588, 0.026492666453123093, -0.17433050274848938, -1.203764796257019, 0.05760483071208, -0.501061737537384, 0.04629756882786751, -0.7360277771949768, -0.1365503966808319, 2.311040163040161, -1.929225206375122, -0.14099302887916565, -0.0011229902738705277, 1.086035132408142, 0.3861798048019409, -0.11901528388261795, 0.4309321641921997, -0.28686782717704773, -0.8225681185722351, -0.5485484004020691, -0.3147471845149994, -0.25481995940208435, 0.2708897590637207, 0.8311757445335388, -1.3381156921386719, -0.4049244523048401, -2.4651312828063965, 0.030823100358247757, -1.2343827486038208, -0.06632733345031738, -2.5774075984954834, 0.6390810012817383, -0.9378758668899536, -0.11722715198993683, -0.1528647243976593, -0.8452152609825134, -0.9313668012619019, 0.20107769966125488, -0.048909991979599, 1.3817750215530396, 0.9943616390228271, -0.3355156183242798, 1.2527717351913452, -1.4321166276931763, 2.6922712326049805, 0.14346623420715332, -0.018285667523741722, 0.5276114344596863, -0.3625985383987427, 0.9900740385055542, 0.5339334607124329, 0.08438318222761154, 0.057586584240198135, 0.02402256801724434, -0.36654746532440186, -0.015535803511738777, -0.8588825464248657, 0.1164214015007019, 0.7041580677032471, 1.5788912773132324, -0.2792716324329376, 0.13219204545021057, 0.2075173258781433, -1.5597162246704102, 0.21197877824306488, -0.38002529740333557, 0.2685208022594452, -2.6901328563690186, -0.06690511852502823, -0.3361416459083557, 0.1609550565481186, -0.16792050004005432, -1.3011078834533691, -0.6439910531044006, 0.01769229583442211, 1.4792670011520386, -1.1746535301208496, -0.09714391082525253, -0.8635689616203308, -0.5255116820335388, -0.0318819023668766, 0.23115193843841553, 0.09486643224954605, 0.007410236168652773, -0.8127411007881165, 0.3279743790626526, 1.361711859703064, -3.297045946121216, 0.6891904473304749, 0.28096136450767517, -0.5742227435112, 0.2201797217130661, -0.800375759601593, -0.8414097428321838, 0.02980169467628002, -0.675274133682251, 0.38447490334510803, 0.15008427202701569, -0.6772633194923401, 0.0629519373178482, 0.11614436656236649, 0.06442002207040787, -0.2840608060359955, 1.5553749799728394, -0.26818206906318665, 0.12322317063808441, 0.10384994745254517, -0.08869190514087677, 0.310678631067276, 0.6218013167381287, 0.005107907112687826, -0.3944956660270691, -0.29529136419296265, -0.127451092004776, 2.3153131008148193, -0.3350301682949066, 1.2652658224105835, 0.7445968985557556, 1.3003510236740112, 1.908636450767517, -0.8149299621582031, -0.43802526593208313, -2.200319766998291, 0.11125610768795013, 0.4419090747833252, -0.01240407582372427, -0.532102644443512, 1.5869150161743164, -0.3003932535648346, 0.9848763346672058, 0.7785760164260864, 0.016415955498814583, -0.4483239948749542, 0.3023339807987213, 0.9484831094741821, 1.9153074026107788, -2.930743455886841, -0.1660115122795105, -0.1431882381439209, -1.1265299320220947, 0.16662552952766418, 0.30669406056404114, 1.4056727886199951, -0.0024270557332783937, -0.0046921479515731335, -0.07939763367176056, 0.0023642450105398893, 0.03768298402428627, -0.7257198691368103, -1.8590189218521118, 1.2980670928955078, -0.41279399394989014, -0.06886684894561768, -0.5818161964416504, 0.05861392989754677, 0.17015253007411957, -0.13424120843410492, 0.37242987751960754, -3.2535111904144287, -0.02845965325832367, -0.11209293454885483, -0.001681320951320231, 0.013215347193181515, -0.05603402853012085, -0.14270958304405212, -0.043848808854818344, -0.6077796220779419, 0.038503680378198624, -0.2210886925458908, -0.12051273882389069, -0.7256945371627808, -0.865921139717102, 0.8571483492851257, 0.8175960183143616, -0.017040370032191277, 0.0016902555944398046, 0.012500971555709839, -0.07380285114049911, -1.3545116186141968, 0.46346986293792725, -0.26912543177604675, -0.09163722395896912, 0.1571849286556244, 0.13673433661460876, 0.3751105070114136, 0.07992221415042877, 0.0609421581029892, -0.13511423766613007, -0.021724490448832512, 0.06260617077350616, 0.7350373268127441, -1.4139046669006348, -0.061952751129865646, -1.4026193618774414, -0.3616484999656677, -0.4154971241950989, 0.7847073078155518, 0.4559997618198395, -0.7633161544799805, -0.09181974828243256, 0.08765772730112076, -0.858556866645813, -0.2954276204109192, -1.2752087116241455, 0.12377041578292847, 0.38528406620025635, -0.15924477577209473, -0.8897595405578613, -0.1505921334028244, 0.9529780745506287, 0.4426155388355255, 0.13234096765518188, -0.30279219150543213, -0.21636991202831268, -1.4570430517196655, 0.008385006338357925, -1.2803641557693481, -0.16502916812896729, -0.29928508400917053, 0.21344977617263794, -0.051447805017232895, -1.3137394189834595, 0.5988616347312927, -0.7642681002616882, -0.729943037033081, -0.3310275077819824, 1.1302815675735474, -0.2524041533470154, 0.3005910813808441, -0.05438677966594696, -0.014656003564596176, 0.11907100677490234, 0.39044758677482605, -0.03611207380890846, -0.30820828676223755, -0.8376159071922302, -0.3108676075935364, 0.36286023259162903, 0.7207099795341492, -1.9154917001724243, 0.1805313527584076, -0.9656195044517517, -0.2588751018047333, -0.016438033431768417, -0.44826316833496094, 1.046697974205017, -0.40458449721336365, -0.9122986793518066, 0.1407719999551773, -0.3752632141113281, 0.7675805687904358, -0.3218279778957367, -0.11991091817617416, -0.049331121146678925, 0.38256508111953735, -0.002541160210967064, 0.9653713703155518, -1.07370924949646, -0.6479380130767822, 0.4484478831291199, -0.02159038931131363, 1.062992811203003, 1.3768131732940674, 0.21848823130130768, -1.7916913032531738, 0.29173895716667175, -0.5477767586708069, -0.3613170385360718, 0.011283687315881252, -0.37507399916648865, -0.11832994222640991, -0.5507077574729919, 0.25441446900367737, -0.1463884711265564, 3.265136480331421, 1.3189090490341187, -0.17526577413082123, 0.5981943011283875, -0.1951545625925064, 0.030355535447597504, -0.91230708360672, 0.4085589051246643, -0.3804381787776947, 1.047250747680664, 0.9799575209617615, -0.12284877896308899, -0.8151715993881226, -1.7954034805297852, 0.5376906394958496, 0.009025669656693935, -0.2862607538700104, -0.03777541220188141, 1.6244044303894043, -0.4939345121383667, -0.2130408138036728, 0.6336671710014343, 0.6413924098014832, -0.08863385766744614, 0.35392332077026367, 0.18039776384830475, 1.2687610387802124, 0.1068885326385498, -0.16184782981872559, 0.10069659352302551, -0.35156673192977905, 0.4031928479671478, 0.8477997183799744, -0.6484130620956421, -0.0645817369222641, 0.0032029191497713327, 0.5345690250396729, -0.6844837665557861, -0.13177767395973206, 0.3062973916530609, -0.006499000824987888, -0.36011815071105957, 0.5110759139060974, -0.09811858832836151, 0.029313787817955017, 0.00011776280007325113, -0.6409444808959961, 0.20251356065273285, 0.11332720518112183, 0.2014663964509964, 0.09056627750396729, 1.235255241394043, -0.03776838257908821, 0.04869299754500389, 0.07521224021911621, -0.31108832359313965, -0.049651797860860825, -0.2576739490032196, 0.012759042903780937, 0.32327696681022644, 0.10442753881216049, -0.08461673557758331, -0.30875274538993835, -1.780310034751892, -0.6109544038772583, -8.28868293762207, -0.39929378032684326, -0.8506508469581604, -0.25803205370903015, -0.012303325347602367, -0.6829302906990051, 0.3884567618370056, 2.1745927333831787, -0.37853991985321045, -0.28566861152648926, -0.3048345148563385, -0.6994994878768921, -0.4347068667411804, 0.33856001496315, -0.006235339678823948, 0.05132008716464043, -0.2676220238208771, -1.2869144678115845, -2.6170949935913086, -1.628816843032837, -1.3283902406692505, -0.15798458456993103, -0.2714023292064667, 2.7011027336120605, 0.3442976474761963, -0.16383445262908936, 0.035880863666534424, -0.4123298227787018, -1.0451618432998657, 0.053040336817502975, -0.10152044892311096, 1.7621560096740723, 0.038330093026161194, -0.3484497666358948, -0.11193352192640305, -0.23434768617153168, 0.2163098305463791, 0.10533120483160019, 0.4843187928199768, -1.7220197916030884, -0.1855144500732422, 0.006766930688172579, 0.07753537595272064, 0.1902294158935547, -0.06743898242712021, -1.3292934894561768, -1.3735733032226562, 0.1583525836467743, 0.7249045968055725, 0.0147093515843153, -0.07321315258741379, -0.1863570660352707, -0.539273202419281, -0.09246477484703064, 1.0944533348083496, -0.6036939024925232, -1.2994706630706787, -0.09616853296756744, -0.04963485524058342, -0.025222470983862877, 3.4951822757720947, -0.2428642064332962, 2.4547600746154785, -0.7279930710792542, 1.588215708732605, 1.3326579332351685, -2.113102912902832, -0.2851056158542633, 0.043392471969127655, -0.16836518049240112, 1.079994559288025, 1.2885411977767944, -0.4607912600040436, -0.2134399712085724, -0.08345126360654831, 1.5940818786621094, -2.3509976863861084, -1.6019282341003418, 0.2709743082523346, 0.08308879286050797, 0.6403289437294006, -1.1083341836929321, -1.1222388744354248, -0.06507855653762817, 0.9255900979042053, -0.5800682902336121, 0.19160370528697968, 0.026040663942694664, 0.18196514248847961, -0.4752030074596405, 0.09386849403381348, -1.3596808910369873, 0.17664559185504913, 1.379009485244751, 0.07456313073635101, -1.1423602104187012, -0.4036540985107422, 0.2936609089374542, 2.2147223949432373, -2.231449604034424, 0.10299696773290634, 0.7013934254646301, -0.00023771838459651917, -0.13167519867420197, 0.04298022761940956, 1.4889012575149536, -1.8124825954437256, 1.4623777866363525, -2.051790237426758, 1.0447049140930176, 1.242018222808838, 2.156761884689331, 0.7098538279533386, -0.1663854718208313, -1.131796956062317, 0.046522319316864014, 2.0575413703918457, -0.07907774299383163, 1.6335783004760742, -0.1880142241716385, 0.046902626752853394, -5.1227192878723145, -1.047373652458191, 0.8856244683265686, -0.15097881853580475, 0.06785915046930313, -1.0428270101547241, -0.42431503534317017, 0.0656588152050972, -0.3091847002506256, -0.05536246299743652, 0.07051593065261841, -0.09324787557125092, 0.23977155983448029, 0.6438882946968079, -1.1431390047073364, 0.31681695580482483, 0.007759410422295332, -0.020320814102888107, 0.27325186133384705, 0.02766452729701996, -1.361498475074768, 0.04063524678349495, -0.13739332556724548, 0.1751827597618103, -1.646064043045044, 0.29089900851249695, -1.927944540977478, -0.36129826307296753, -0.26929253339767456, -0.1030692532658577, -1.9438272714614868, 0.967556893825531, -0.1398777812719345, -0.023977279663085938, 0.08595388382673264, -0.16159161925315857, -1.7142443656921387, 0.15195603668689728, -0.002075020456686616, 0.27413734793663025, 0.05365854501724243, 1.5339701175689697, -1.2284079790115356, 0.49827101826667786, 0.07757092267274857, -0.027460327371954918, 0.3701607882976532, 0.11829999089241028, -2.5706496238708496, -0.45720434188842773, 0.3567785620689392, -0.3085392713546753, -0.38153862953186035, 0.013194828294217587, -1.2319453954696655, -0.6490898728370667, 0.29364174604415894, 0.011539959348738194, -0.9573166370391846, -0.6428076028823853, -0.11378955841064453, -0.013003876432776451, -2.4247779846191406, 0.428396612405777, -0.46173161268234253, 0.47941726446151733, 0.3189273774623871, 0.09711237996816635, 0.574242115020752, 0.07297037541866302, 0.028306500986218452, -0.35841527581214905, -0.3419039845466614, 0.24784643948078156, -0.003749953815713525, -2.4901926517486572, 2.6630783081054688, 0.6155497431755066, -0.04614465683698654, 0.09047450125217438, 0.031040167436003685, 0.18422697484493256, 0.9220383167266846, -0.3773178458213806, -0.261981338262558, -0.01362006925046444, -0.08116655051708221, 1.8112008571624756, -0.03712508827447891, 0.8026925921440125, 0.33524587750434875, -2.112009286880493, 0.05417249724268913, -0.36647266149520874, 0.15912273526191711, 0.27525028586387634, 0.08998646587133408, -0.5959821343421936, 1.4976779222488403, 0.04834708198904991, -0.11659207940101624, -0.2904496192932129, 0.44119834899902344, 0.2714385688304901, -0.13968151807785034, 0.42472338676452637, -0.2841034233570099, 2.4854021072387695, -0.027726616710424423, -0.03787611424922943, -1.9073128700256348, 0.4208918511867523, -0.3526584506034851, 0.49592065811157227, -0.7961130738258362, -1.0213510990142822, 3.6195998191833496, -0.7574254274368286, -0.16303344070911407, 0.26081719994544983, 3.052523374557495, 0.3541513979434967, 0.03054349683225155, -0.07804711908102036, 1.6685049533843994, 1.2979384660720825, -0.06004786491394043, -0.14871877431869507, 0.05326829478144646, -1.6431668996810913, 2.0074596405029297, 0.4627361297607422, -1.2381395101547241, -0.21493807435035706, -0.1064230278134346, 0.8223194479942322, 1.9276331663131714, 0.005040407180786133, -0.16405010223388672, -0.8184516429901123, -0.11713045835494995, 1.7368018627166748, -0.04859429597854614, -0.6951950192451477, 3.069782257080078, 1.8461437225341797, 0.41100001335144043, 0.25206732749938965, -1.681107997894287, 0.010033771395683289, -0.7865411639213562, 0.2215346097946167, 0.012866174802184105, 1.5280406475067139, -0.1491154432296753, -0.04718800634145737, -0.06553936749696732, -1.0188977718353271, 1.2448893785476685, -0.677280604839325, -0.15988364815711975, -0.021855337545275688, -0.06262524425983429, 0.016411373391747475, 0.41459721326828003, -0.10858912020921707, 0.19378016889095306, 0.6685419082641602, -0.8511869311332703, 0.5354660153388977, 0.15756216645240784, 0.6966666579246521, 0.06775129586458206, -1.5756756067276, 0.0038864745292812586, -0.02349132113158703, 0.007336820475757122, 2.1476876735687256, 0.9361344575881958, -1.2478604316711426, -0.5065733194351196, 0.07906042784452438, 3.776251792907715, 0.8339428901672363, -0.5416820049285889, 1.9475351572036743, -0.3871249258518219, 0.3880552351474762, -0.8501878976821899, 0.03019092231988907, -2.134648323059082, 0.14785327017307281, -0.27888524532318115, 2.0434248447418213, -0.0960949957370758, 1.0960266590118408, 0.027048850432038307, 2.311298131942749, 0.2033436894416809, -0.09294950217008591, -0.3092045187950134, -0.0023252915125340223, -0.0180271714925766, -0.018577655777335167, 1.477407693862915, -0.834614634513855, -1.8028267621994019, 0.5187155604362488, 0.012678820639848709, 0.05347428843379021, -0.0035565721336752176, 3.3293302059173584, 0.1462409496307373, -0.053135767579078674, -1.3440654277801514, -0.49384891986846924, -1.783462643623352, 0.26874083280563354, -0.8496582508087158, 0.21392279863357544, 0.2554275393486023, 0.05343730002641678, -0.5058171153068542, 0.29378384351730347, -0.20203183591365814, -0.0009040849981829524, 0.002943715313449502, 0.2643285095691681, -0.25126785039901733, -0.46539536118507385, 0.008117570541799068, -0.4556909203529358, 1.715055227279663, -1.4459925889968872, -0.9379404187202454, -0.518389105796814, -0.37380650639533997, -0.8635502457618713, 0.17886720597743988, -0.18625521659851074, 0.13410255312919617, 0.036615680903196335, 0.3465389609336853, -0.19358423352241516, 0.636370837688446, 0.01202146802097559, 0.356299489736557, 0.10728755593299866, 0.4718398153781891, -0.13564951717853546, -0.32235053181648254, -0.10313724726438522, 1.1926077604293823, -0.35378050804138184, -0.6509931087493896, 0.9517324566841125, 0.32634997367858887, -0.23259134590625763, -0.3708019554615021, -0.0022585575934499502, 1.1067836284637451, 0.017018087208271027, -0.36106422543525696, 0.40387165546417236, -0.702855110168457, 0.6917195916175842, -3.671116590499878, -0.5035145878791809, 0.01555416826158762, -1.7188305854797363, -0.5698722004890442, -0.21897418797016144, -0.7332019805908203, 0.2237221598625183, 0.4454857110977173, -0.07264935970306396, 0.43012964725494385, 0.10378550738096237, 0.3177136182785034, -0.9044971466064453, -0.4480436146259308, -1.5684168338775635, 0.019616959616541862, -0.023439256474375725, -1.256367802619934, 0.21895849704742432, 0.06328967958688736, -0.15251052379608154, 0.2323683500289917, -0.20344072580337524, 0.8756564855575562, 0.3271501660346985, 0.1459202617406845, 1.0582339763641357, 0.1613471508026123, -0.28815868496894836, -1.7875609397888184, -0.6498172879219055, -0.0661245733499527, 3.062574863433838, 0.03872283175587654, 0.1640571802854538, 0.5982757210731506, 0.2582751214504242, 0.46693289279937744, 0.4910789430141449, -0.6825776696205139, -0.3503677248954773, -0.2005687654018402, 0.1731308102607727, 0.12071410566568375, 0.048336323350667953, -0.25124669075012207, 0.26025059819221497, -0.36233580112457275, 3.5617692470550537, -0.7555413842201233, -0.33929046988487244, 0.10592473298311234, 1.537684679031372, 2.1157352924346924, 0.3401740789413452, 0.7321476340293884, 0.45393526554107666, -0.3298399746417999, 0.0540907196700573, 0.09860040247440338, 1.6407036781311035, -0.1808517724275589, -1.4676613807678223, 0.15009917318820953, 0.24120494723320007, -1.0880870819091797, -1.2616084814071655, -0.06140300631523132, 1.0060665607452393, -0.14832860231399536, 0.3279644548892975, -0.4842461049556732, -1.3870691061019897, -0.31156450510025024, -0.9239217042922974, 0.8499153852462769, -1.170211911201477, 0.9732604026794434, 0.18138840794563293, -1.2385629415512085, -0.2944829761981964, -0.0344243086874485, 0.06823211908340454, 0.1689862608909607, -2.9145359992980957, -0.3488236665725708, -0.04665062576532364, -1.1973987817764282, -6.18540620803833, -0.03738992661237717, 0.014831248670816422, -0.11635806411504745, 1.7121601104736328, 0.40028220415115356, 0.11974135041236877, -0.13859540224075317, -4.230443477630615, -1.9603204727172852, -1.9501296281814575, -0.1696317046880722, -0.18665115535259247, 0.2347579449415207, 0.31077033281326294, -0.0772695392370224, -0.1375763863325119, 0.3363533020019531, -0.04352201148867607, 0.22453273832798004, -0.34359484910964966, 0.45021799206733704, 0.8559959530830383, 0.9256070256233215, -0.28375789523124695, 1.6315760612487793, -0.021935641765594482, -0.0290992371737957, 0.47201603651046753, -0.9124346971511841, -0.013488257303833961, 0.041257381439208984, -0.5632959604263306, -0.9680514335632324, 2.214270830154419, 0.029158681631088257, 0.5266585350036621, 0.008210878819227219, -0.3993482291698456, -0.0655192881822586, -2.1896395683288574, -1.2935469150543213, -0.0018419044790789485, -0.0015446426114067435, 0.19463758170604706, -1.1442102193832397, 0.14747510850429535, -1.255229115486145, -0.9455277919769287, -0.16430245339870453, -0.1288495659828186, 0.6658864617347717, -1.3628987073898315, 3.301851749420166, -0.07061775773763657, -0.0958046242594719, 0.049278099089860916, 0.47883859276771545, 0.34150439500808716, 0.06379437446594238, 0.021738838404417038, 0.011908994987607002, -1.08005690574646, 0.30438920855522156, 0.8969734311103821, -0.2507018446922302, -0.00254327361471951, 1.8882226943969727, 0.6731627583503723, -0.10728553682565689, 0.4901602864265442, 0.08981100469827652, 0.0005613794783130288, -0.1685163527727127, -1.5386598110198975, -0.031814612448215485, 0.5050576329231262, 1.7045694589614868, -0.06193700432777405, 0.041542988270521164, -0.39976513385772705, 0.14376330375671387, 0.7456867098808289, 0.06492967903614044, 0.10564567893743515, -0.967819333076477, -3.040315628051758, 1.5347665548324585, 1.7505381107330322, 1.1933250427246094, 1.7297924757003784, 1.6519556045532227, -0.1605941355228424, -0.06837386637926102, 0.19523440301418304, -0.7202840447425842, 0.9302871227264404, -1.245404601097107, -1.9673916101455688, 0.11046085506677628, -1.1015346050262451, 0.6401768922805786, -1.1099474430084229, -0.4249418079853058, 0.1218080073595047, -0.5900309085845947, 1.0939254760742188, 0.062375254929065704, -1.1117987632751465, -0.03419351950287819, -0.045691050589084625, 0.2080889344215393, -1.7595425844192505, -0.07985930889844894, -0.994017481803894, -1.296849250793457, -0.45311272144317627, 0.23198528587818146, -0.0492546409368515, -0.7709631323814392, 1.0365208387374878, 0.3678591251373291, -0.6690003275871277, 1.6438615322113037, -0.018510952591896057, -0.8562185764312744, 1.931546688079834, 0.1401495337486267, -0.9358503222465515, 0.32888293266296387, 0.5260652899742126, 1.0221220254898071, 0.43255704641342163, -0.06122664362192154, -0.558428943157196, 0.6455190777778625, -0.8187049031257629, 0.051370881497859955, 0.2893334925174713, -0.10787393152713776, 0.4182139039039612, 2.859734296798706, 0.054178427904844284, 0.0013450535479933023, 0.512110710144043, -0.22279611229896545, 0.2048514187335968, 0.9372748732566833, 0.07355944812297821, -0.5820653438568115, -0.6092366576194763, -0.027313007041811943, -0.6561945676803589, 0.9573096632957458, -1.881496787071228, 0.22060854732990265, -0.4826820492744446, -1.378614068031311, -0.08652079850435257, -0.024099135771393776, -0.035674747079610825, -1.4489580392837524, -0.5258433818817139, -4.0589599609375, -0.009833434596657753, 0.061567503958940506, -0.010034502483904362, 0.5199615955352783, 0.08611352741718292, 0.17561812698841095, 0.8111667633056641, 0.3294054865837097, -0.13740667700767517, -1.0074968338012695, 1.6414992809295654, 0.9397549629211426, 1.0621187686920166, -0.8519885540008545, 0.27479955554008484, -1.3677526712417603, 0.001837490708567202, -0.9811764359474182, -0.31024911999702454, 0.17787809669971466, 0.15340101718902588, 0.7990965843200684, 0.035541780292987823, -0.025363348424434662, -1.3131396770477295, 0.16759131848812103, 0.25372833013534546, 0.2554604709148407, 0.1114867553114891, 0.9166776537895203, 0.47853347659111023, -1.0136258602142334, -0.9266499876976013, 0.3875582218170166, -0.07007478922605515, 0.4454469084739685, -1.9230539798736572, -0.07611404359340668, 0.811835765838623, -0.19732165336608887, 0.03375222161412239, -0.9613831043243408, 1.0764561891555786, 5.296332836151123, -0.5387073755264282, 0.29421254992485046, 1.4643580913543701, -2.118035078048706, 0.44451165199279785, -0.5603979825973511, -0.5607231259346008, 1.121781349182129, 0.17266057431697845, -0.4557543098926544, 0.30630648136138916, 4.740457057952881, -0.23151150345802307, 0.031067898496985435, -1.2925533056259155, 0.15834476053714752, 0.06756149232387543, -0.013537727296352386, -0.7874106168746948, 0.9911842346191406, 1.2342277765274048, -0.6216669678688049, 0.1849777102470398, 0.0834738090634346, 0.6330086588859558, -0.20882466435432434, 0.4648386836051941, -0.2501451075077057, 0.0029039448127150536, 1.468387484550476, -0.8412243723869324, -0.42607083916664124, -0.015203911811113358, 0.04712367430329323, -0.2665052115917206, -0.49840110540390015, -0.07949166744947433, -0.021027328446507454, 1.408186912536621, 2.1691675186157227, 0.4569871723651886, -4.7031121253967285, -0.6644914746284485, 0.6459800004959106, 0.02541373111307621, -0.977649450302124, -1.2077692747116089, -0.40904876589775085, 2.632424831390381, 0.006666197441518307, 0.26565563678741455, -0.02288055792450905, 0.06316234916448593, 0.031284332275390625, -0.33005377650260925, 0.10576728731393814, 4.370297431945801, 1.0565354824066162, 0.25318723917007446, 0.5769244432449341, 0.2528182864189148, 1.531701922416687, 0.01656462624669075, 1.5389424562454224, -1.3346935510635376, -0.38478994369506836, 0.1247272863984108, -0.07216107100248337, 0.2319764345884323, -1.084223985671997, -0.8388881087303162, -0.8285730481147766, 0.7631329298019409, 0.3839092254638672, 0.46480998396873474, -0.7105981111526489, 1.9799885749816895, 0.031366243958473206, -1.2882720232009888, 1.8977586030960083, -0.42874157428741455, -0.11676064878702164, 0.033483896404504776, 0.25686362385749817, -0.2818460762500763, -0.07711751759052277, 0.21925093233585358, 0.8047012686729431, 0.07944377511739731, 2.461240530014038, -2.5438644886016846, -0.5621029138565063, -0.4757181406021118, 0.059460222721099854, -0.6151522994041443, 0.2986329197883606, -0.3213595449924469, -1.699405312538147, -0.06645052134990692, 0.947105884552002, 1.1994783878326416, 1.4437153339385986, 0.017246365547180176, 0.2985703647136688, -1.6750329732894897, 0.44428306818008423, 0.19512119889259338, 0.03453104943037033, 0.06036882475018501, 0.0351143404841423, 0.23643885552883148, 0.9863108396530151, 0.6369155049324036, -0.4109014570713043, 0.01499424409121275, 0.9639419913291931, 0.8265923261642456, -1.6270076036453247, 0.18430384993553162, -0.09585011005401611, -0.20223067700862885, -0.32286572456359863, -0.4155806005001068, -0.39163580536842346, 0.2000308632850647, 0.18597392737865448, 0.07364450395107269, 0.9105868339538574, -1.362401008605957, 0.08520375937223434, 0.262259840965271, 0.023034563288092613, 0.27092480659484863, 0.6667935252189636, 0.4459265172481537, 2.1900787353515625, -0.6726588606834412, -0.4343729615211487, 1.021934986114502, -0.14336083829402924, -0.2834629416465759, 0.29080459475517273, -0.675295352935791, 1.6010464429855347, 0.6354190111160278, -0.026142543181777, 0.023120392113924026, -0.37497448921203613, -0.7939366102218628, -1.2480682134628296, -0.9042311310768127, 0.2154533565044403, -2.2951273918151855, -1.2842540740966797, 0.05257025733590126, 0.03778357431292534, 1.9249777793884277, -0.2441231906414032, 0.35351648926734924, 0.14228986203670502, -0.44331085681915283, -0.42030906677246094, -0.007007676642388105, -1.5939472913742065, -0.12418927252292633, -0.2846173644065857, -0.4594472348690033, 0.07057847827672958, 1.0644543170928955, -1.3033314943313599, -0.18792766332626343, -0.21728749573230743, -0.36948731541633606, 0.26509663462638855, -0.17811961472034454, -0.6277005076408386, 0.5311839580535889, 0.06152942776679993, -0.17915743589401245, 1.1124653816223145, 0.11612645536661148, -0.01492274645715952, -0.6771395802497864, 0.38583141565322876, 0.31308498978614807, -0.6274411678314209, -0.6089567542076111, -1.1180778741836548, -0.31996431946754456, 0.5742764472961426, -0.5612756609916687, -0.3783348500728607, 3.8246910572052, 0.27550560235977173, 0.3107551038265228, -0.8771690726280212, -0.34899839758872986, 0.447460800409317, -1.9576787948608398, 3.3702147006988525, 0.6009458303451538, 0.17617642879486084, -0.01573021709918976, 1.0816203355789185, 0.3333508372306824, -0.9640851020812988, 1.139029622077942, 0.04895557090640068, -0.4233452379703522, -0.08380062133073807, -0.271885484457016, 0.030552953481674194, -0.34126946330070496, 0.08020887523889542, -0.7252256870269775, 1.2946480512619019, -0.10862071812152863, 0.5251412987709045, -0.014219558797776699, -2.246901273727417, -0.1311497688293457, 0.3221965730190277, -0.08619426190853119, 0.3146286606788635, 0.2825373709201813, -0.06387900561094284, 0.957108736038208, -0.42184212803840637, -0.24864134192466736, 0.022286484017968178, 0.012399421073496342, 0.5868579149246216, 0.2124173790216446, -0.5963113903999329, 0.02558138221502304, -0.0016595239285379648, 0.3851136863231659, -1.7939804792404175, -1.1322041749954224, 0.1003963053226471, 0.11510760337114334, -1.188439130783081, -0.0206745732575655, -1.077589750289917, 4.133659839630127, 0.4594825506210327, -0.07233979552984238, 0.6310055255889893, 0.05803772434592247, 0.2559569478034973, -0.38491740822792053, -1.085726022720337, 0.022151460871100426, 0.046632878482341766, -0.28093475103378296, 0.3458406925201416, 0.2557227611541748, 3.1085753440856934, 1.2920620441436768, 0.35083773732185364, 0.8200870156288147, -0.6826351284980774, 0.1364368200302124, -0.8265756368637085, 0.3138260841369629, -1.3083359003067017, 0.03177972137928009, -0.2734008729457855, -0.4974522292613983, -1.3999781608581543, -0.16604609787464142, -2.4457359313964844, -0.7678223252296448, -0.004643999505788088, -0.6637213230133057, -1.8689730167388916, 0.0021862201392650604, 0.05990948900580406, 0.4334481954574585, 0.8042217493057251, 1.8950912952423096, -0.4745420217514038, 0.03080013580620289, 0.030459396541118622, -0.154812753200531, -0.5209963321685791, 0.5434687733650208, -0.7284637093544006, 0.7130709290504456, -0.12302938103675842, -0.3290618658065796, -0.040129683911800385, 0.43363866209983826, 0.8030613660812378, -0.8200260400772095, 0.07200855761766434, 2.362171173095703, -0.149943009018898, -0.887709379196167, 0.7427279949188232, -0.1801283359527588, 0.0690217986702919, 0.2820912301540375, -0.9885390996932983, -0.4516884386539459, -0.0020089761819690466, -0.4464295506477356, 0.012693052180111408, -0.029995733872056007, 0.5803156495094299, 1.834587574005127, 0.26406344771385193, 0.23237542808055878, -0.5861032605171204, -0.0006834972300566733, 1.0240662097930908, 0.8556249141693115, -0.07791963219642639, -0.2644663453102112, -1.439583420753479, 0.3902125060558319, 0.0449480265378952, 0.3678210973739624, 0.14725305140018463, -0.05070394650101662, 0.38094857335090637, -0.04909172281622887, -0.09508534520864487, 0.5633931756019592, -0.22208641469478607, 0.17668820917606354, -1.6971898078918457, 0.3589434027671814, -0.1642637401819229, -0.7159914374351501, -1.2422733306884766, 0.23580357432365417, 0.04092950001358986, 0.19477926194667816, 0.14383713901042938, -3.716747999191284, -0.7603777647018433, 0.03854891657829285, -0.1808687299489975, -1.8974013328552246, 0.9047867655754089, 0.23427653312683105, -0.6859582662582397, 0.5182560086250305, 0.8220816850662231, -0.39629724621772766, 1.3727965354919434, -0.8630782961845398, 0.11951521784067154, -0.3483266830444336, -1.192589521408081, -0.24011549353599548, -1.0382133722305298, -0.24603214859962463, 0.3734588623046875, 0.022186871618032455, -2.3607048988342285, -0.15027058124542236, 1.4837007522583008, 0.3863404393196106, -0.3655511736869812, -2.066370725631714, 0.3683447241783142, 0.6598544716835022, -0.00041399625479243696, 1.0491359233856201, 0.01421383861452341, 0.5393797159194946, 0.30769509077072144, -0.913488507270813, 0.15268774330615997, 0.03519389405846596, 0.9241741299629211, 0.184278666973114, -0.6245213747024536, -0.15208777785301208, 0.007631329819560051, -0.4337424337863922, 0.19436496496200562, -2.281935453414917, -1.097256064414978, -0.5907682180404663, 0.49210765957832336, -0.058892279863357544, 1.3599220514297485, -0.009176979772746563, -0.2989504337310791, 0.332324743270874, 1.0677486658096313, -0.03400176763534546, 0.24150410294532776, -1.2142549753189087, 1.0358635187149048, -1.035469889640808, 0.049812790006399155, 0.4944539964199066, -3.722001791000366, -0.18850001692771912, -0.8937057852745056, 3.205237865447998, -0.024319540709257126, 0.14016547799110413, -0.3659885823726654, -0.16354545950889587, -0.4707680344581604, 0.23864112794399261, 0.30006375908851624, 0.09503504633903503, 0.021069204434752464, -5.08035135269165, 0.2840140759944916, -0.8670126795768738, -0.7889785170555115, -0.3919931650161743, 0.27793872356414795, -0.8683323264122009, 0.2657500207424164, -0.014952617697417736, 0.04397917538881302, -0.7463690638542175, 1.1877492666244507, -0.05425868183374405, 0.07929911464452744, 0.11941830813884735, -0.3521270155906677, -0.11922425031661987, -0.7061379551887512, -1.4125869274139404, 0.2869071960449219, -1.0609482526779175, 0.16117393970489502, 0.5783958435058594, -0.06592843681573868, -0.0318249836564064, -0.28406965732574463, 0.30375510454177856, 0.09812845289707184, 0.019584832713007927, 0.23609286546707153, -0.18446047604084015, -1.3528118133544922, 0.5106634497642517, 2.504446506500244, -1.0352951288223267, -0.19099482893943787, 0.5864097476005554, 1.0804510116577148, 0.34204229712486267, -3.7700109481811523, 0.44629886746406555, 1.002446174621582, 0.04126788675785065, 0.6826443076133728, -0.3334116041660309, -0.6119229197502136, -0.8768991231918335, 0.9735734462738037, -0.21854054927825928, -0.00011716562585206702, 0.004338392056524754, 0.08303380012512207, 0.43318259716033936, 1.4008935689926147, 0.021572276949882507, -0.4547627866268158, 0.47796639800071716, -0.06010933965444565, 0.3680601716041565, 1.499233603477478, 0.240497425198555, -0.009571035392582417, -0.811832070350647, 0.8275852203369141, -2.1260299682617188, 0.34145647287368774, -0.28335270285606384, 0.5376688241958618, 0.25803324580192566, -0.226896271109581, 0.006593511905521154, 0.25145667791366577, 0.03665672615170479, -0.05189311131834984, 1.683755874633789, 0.1262248009443283, 0.11100396513938904, 0.24896815419197083, -0.6171684861183167, 0.44185879826545715, -0.017806323245167732, 1.3629518747329712, 0.14996765553951263, 1.3356724977493286, 0.5132821202278137, 0.016675759106874466, -0.13517892360687256, -0.13288778066635132, -1.0415434837341309, 0.10113242268562317, 0.21060730516910553, 0.4231283366680145, -0.33661800622940063, 0.14468112587928772, 2.814819812774658, -0.37222298979759216, 0.5533525943756104, 2.854410409927368, -0.06522232294082642, -2.8291637897491455, -1.2888849973678589, 0.21130971610546112, 3.459657669067383, 0.5898748636245728, 0.0965426042675972, -3.7510859966278076, -0.005218640435487032, -0.2397056370973587, 2.2207424640655518, -0.27211645245552063, -0.4674561619758606, -1.7224699258804321, 0.690794825553894, 0.17113658785820007, -0.038405053317546844, -0.31041932106018066, -0.17774111032485962, -0.08205444365739822, -0.8096784949302673, 0.03494413569569588, 0.24883732199668884, 0.03448523208498955, 0.07797683030366898, 0.09163010865449905, -0.6930851340293884, 0.9066282510757446, 1.462283968925476, -2.2127010822296143, -0.4358389377593994, -0.23540803790092468, 0.9113687872886658, 1.2036011219024658, 1.5954099893569946, -0.6885414123535156, 0.0354941189289093, -0.00620981166139245, 0.18023334443569183, 1.625236988067627, 0.2645898759365082, 0.04150955751538277, -0.5825324654579163, 1.69051194190979, -1.4029154777526855, 0.36298033595085144, 0.3942216634750366, 0.42820823192596436, 0.1243228167295456, 0.16791236400604248, 0.122206911444664, -0.6455062031745911, 1.0162041187286377, -0.17672692239284515, -0.18421560525894165, 0.021385710686445236, -0.4460179805755615, -0.6288080811500549, -1.6099648475646973, 0.5247430801391602, 0.006028817035257816, -0.6382986903190613, 0.40891629457473755, 0.3345278203487396, 2.0861592292785645, -1.134275197982788, 2.425189256668091, 0.1968431919813156, 0.7815103530883789, -1.341844916343689, -0.10235519707202911, 1.1304576396942139, 0.8483572602272034, -0.8118789792060852, -0.23867176473140717, 0.11630024760961533, 0.12305480986833572, -0.49545568227767944, -0.4573417901992798, -2.407555103302002, -0.28870105743408203, -0.9208247065544128, 0.040644846856594086, 0.32925519347190857, 0.14591529965400696, 0.7335055470466614, 2.3976798057556152, 0.35865163803100586, -0.441260427236557, -0.07869157940149307, 2.289163112640381, 0.23474912345409393, 0.6956981420516968, -0.24508275091648102, 0.001813608454540372, -4.183583736419678, 0.6480379104614258, 0.009464715607464314, 0.3483841121196747, -0.0857204869389534, -0.550759494304657, 0.05226362869143486, -0.784464418888092, 0.1898837685585022, -0.550940215587616, -0.13321825861930847, -0.20638681948184967, 0.2950948476791382, 0.270429790019989, -0.8847017884254456, 0.10632291436195374, -4.08989953994751, 0.018992114812135696, 0.1876077950000763, -0.234843909740448, 0.5675785541534424, 0.00702268211171031, -0.7536442875862122, -0.2050757259130478, -0.2866342067718506, -2.213449478149414, 0.11762772500514984, -0.536986231803894, -0.7639166712760925, 0.28412091732025146, 5.6178412437438965, 0.4042009115219116, -0.08086774498224258, -0.41973260045051575, 0.8561577200889587, 0.05628017708659172, -0.4515756070613861, 0.04250703006982803, -0.08678942173719406, -0.17934508621692657, 0.02592751570045948, 0.029164768755435944, -0.051475584506988525, -0.5112526416778564, -1.087814211845398, -0.7996945977210999, -0.050563994795084, 0.34651198983192444, 0.42987215518951416, 1.4811782836914062, -0.8652147650718689, 0.0038037444464862347, -0.533377468585968, 0.05571601167321205, -0.19031985104084015, -0.8363516926765442, -0.5535716414451599, -3.4811086654663086, 1.0368918180465698, -0.2418566793203354, -0.13428010046482086, 0.8152895569801331, 1.7369860410690308, 4.198387622833252, 3.042619228363037, 1.2725616693496704, 0.30421289801597595, -0.3647102117538452, -1.8040800094604492, 0.5431198477745056, 0.7851660847663879, 0.6052052974700928, -0.8864356279373169, 0.03804437816143036, -0.07919492572546005, -0.8961237668991089, 0.2384960800409317, 1.3390535116195679, -0.2310989946126938, 0.08462594449520111, 0.004021139815449715, -0.0025062893982976675, 0.20003531873226166, 0.26166728138923645, 0.28782418370246887, 0.15718023478984833, -4.69100284576416, 0.04021270573139191, -0.8245186805725098, -0.8017125129699707, 0.01637836918234825, 0.23596973717212677, -0.09066490083932877, -0.24335797131061554, -0.2066863626241684, 0.27192750573158264, 1.3466053009033203, 0.07093877345323563, 1.0314031839370728, 1.0244510173797607, 0.04154135659337044, 1.1192913055419922, 0.008753669448196888, 1.4550279378890991, -0.0150603037327528, -0.44632458686828613, 0.3122454285621643, -0.8997830748558044, 0.9980848431587219, 0.1523449718952179, -0.09129861742258072, -0.6035582423210144, 0.2686762809753418, -0.31026262044906616, -0.10878770798444748, -0.5471370816230774, -3.1598076820373535, 0.4511229991912842, -0.004282159265130758, 0.32503485679626465, -0.01615510880947113, 0.060199905186891556, -0.5071941018104553, -0.11791309714317322, -1.1738470792770386, -0.4267818331718445, 0.23510810732841492, 0.2707810401916504, 0.02412775345146656, 0.3355489671230316, -1.8287910223007202, 0.2995089292526245, 1.0237476825714111, 0.042486581951379776, 3.8162384033203125, 0.7939128875732422, 0.3774298429489136, -0.1568956971168518, 0.43253299593925476, -0.7055874466896057, -1.5316561460494995, 0.26346567273139954, 1.3095567226409912, -0.029266998171806335, 1.9043983221054077, -4.958009243011475, -0.016047779470682144, 0.79503333568573, -0.24744917452335358, 0.64530348777771, 5.319184303283691, -0.30312827229499817, -0.06098666787147522, 0.24756382405757904, -0.2212340235710144, -0.8282344937324524, 0.25557923316955566, -0.057909127324819565, -0.05003586784005165, -1.2996952533721924, 0.1811867356300354, 1.132171869277954, 0.07189824432134628, -2.8734233379364014, -0.20515108108520508, -0.030086679384112358, 0.5162662863731384, -0.5684621334075928, -0.7439260482788086, -0.009497543796896935, 0.7133907675743103, -2.2346644401550293, -0.6100345253944397, -0.15577732026576996, 1.4836844205856323, -1.2523486614227295, -0.4336596429347992, -0.21187053620815277, -1.1658940315246582, 0.1451706439256668, 0.06295562535524368, -0.008968395181000233, -0.029958536848425865, 1.1869343519210815, 2.622623920440674, -1.450976014137268, -1.5014679431915283, 0.33880841732025146, -0.2978370189666748, 0.10329452157020569, -0.21949583292007446, -0.004929885268211365, -0.2171856313943863, 0.5592885613441467, -0.23646993935108185, 0.8624351620674133, -0.4648383557796478, -1.5985194444656372, 0.4958617687225342, -2.5354888439178467, 0.3137267529964447, -0.5422510504722595, -0.2666470408439636, 0.0007179970853030682, -0.20615296065807343, -0.04871480539441109, 0.16218873858451843, 1.4945212602615356, 0.40858399868011475, -0.18441006541252136, -0.9483230710029602, -0.4479139447212219, 0.0027152986731380224, 0.0007472953293472528, 0.019765052944421768, 1.476899266242981, -1.7737305164337158, 0.26176002621650696, 0.2883526682853699, -0.7131257653236389, 0.646621584892273, 3.4424943923950195, -0.36804187297821045, 1.3889564275741577, 0.2657890021800995, 0.2848871648311615, -0.5127386450767517, -0.20778952538967133, -1.011792778968811, -0.08845402300357819, 1.789745807647705, -0.10439109802246094, -1.023399829864502, 0.24912552535533905, 0.08044560253620148, -0.5077908039093018, -0.373623788356781, 0.7720475792884827, 0.1892186850309372, 0.04063009098172188, -2.170757293701172, 0.7744901180267334, -0.030423521995544434, -0.13976514339447021, -3.1531174182891846, 0.8397565484046936, 0.18058238923549652, 0.10124047100543976, -0.1895684450864792, -0.6002466678619385, -0.21104863286018372, -0.11853396147489548, 0.785366952419281, -0.4651651680469513, 0.38774174451828003, -1.1846821308135986, -0.4238189458847046, 0.4585881531238556, -1.7032420635223389, 2.353825807571411, -0.5809683799743652, 0.024084072560071945, 0.3172219395637512, -0.6424021124839783, -1.8523433208465576, -1.1702135801315308, -0.7832459807395935, -0.11676665395498276, 0.09915105998516083, -0.0035931484308093786, -1.3118505477905273, -0.032548077404499054, -0.514779269695282, 1.7536799907684326, -0.41535162925720215, -0.33348941802978516, -0.35124608874320984, -0.40321269631385803, 0.25911468267440796, 0.2582154870033264, 1.3821316957473755, -0.2735922038555145, 0.7995187044143677, -1.1344780921936035, -0.05271296948194504, 0.26459407806396484, 1.4353164434432983, 0.6609920859336853, -0.07126668840646744, -0.07447097450494766, -0.33709436655044556, -1.7276042699813843, -0.795738935470581, -0.22241660952568054, 0.28171828389167786, 0.7616555094718933, -1.7704824209213257, 0.6284283399581909, -0.16666460037231445, -0.453689306974411, 0.08989867568016052, -0.5873421430587769, -1.4386647939682007, 0.01137421652674675, -0.0691264420747757, 0.008811939507722855, -0.1978246122598648, -2.790947914123535, -0.22341249883174896, -0.04489840567111969, 0.2926017940044403, -0.365323007106781, 1.4012459516525269, 0.07660257071256638, 0.024759767577052116, 1.8022303581237793, -5.894411563873291, 0.10696659237146378, 0.12374598532915115, -1.4594427347183228, -1.5503212213516235, -0.10602822154760361, -0.13221262395381927, 0.6787919402122498, -0.01157876756042242, 1.409710168838501, 1.0367121696472168, 0.18669891357421875], [-0.4457537829875946, 1.0187392234802246, -0.1214502677321434, -1.2689050436019897, 1.1394487619400024, 0.4769904315471649, 0.027986792847514153, 0.14768652617931366, -0.6982834339141846, -0.02219771035015583, 0.0017547404859215021, 0.8722532391548157, 0.2080611139535904, -0.3054127097129822, -0.10543055087327957, 0.0853022113442421, -0.5423084497451782, -0.5121786594390869, -0.14550797641277313, -0.5878922939300537, 0.18147146701812744, -0.10315932333469391, 0.017468394711613655, 0.47802430391311646, -0.43629753589630127, 0.5609257221221924, -1.3763759136199951, -0.2622663378715515, 0.6053233742713928, -0.09520512074232101, -0.4364898204803467, 0.23632317781448364, 0.19324560463428497, 0.19243909418582916, 0.045988213270902634, -1.2358273267745972, 1.4787448644638062, -0.1288582980632782, 0.8452790379524231, -1.6025606393814087, -0.393446147441864, 0.4542084038257599, 0.33218827843666077, 0.0694105476140976, 0.05587583780288696, -1.127174973487854, -0.4789685308933258, 0.5948746204376221, 0.3629879355430603, 0.45881280303001404, -0.05299638211727142, -0.27342137694358826, -0.08438824862241745, -0.47294965386390686, -0.08667782694101334, -0.9344905018806458, 0.051526546478271484, 0.002433279762044549, 0.03770485520362854, 0.09837176650762558, -0.5421070456504822, -0.0817076712846756, 0.005544794257730246, -1.0460103750228882, -0.10480528324842453, -0.19847458600997925, 0.399039089679718, 0.7258599996566772, 0.031094934791326523, 0.30265554785728455, -0.12277203053236008, 1.3323988914489746, -1.5265893936157227, -0.8015235662460327, -1.1598933935165405, 0.1156674176454544, -0.21439199149608612, -0.8093841075897217, 0.010815576650202274, 0.0680820643901825, 0.3924368917942047, 0.45157867670059204, -0.2960323989391327, 0.1395474672317505, 0.19802355766296387, -0.14340993762016296, 0.03185829147696495, -0.7621785998344421, -1.0116569995880127, 0.9303805232048035, -0.02944132313132286, -0.04491008445620537, 0.2473706603050232, 0.9836651086807251, -0.5035468339920044, 0.9478204250335693, -0.06432077288627625, -0.308016300201416, -0.704237699508667, -0.12400443851947784, -0.056318413466215134, 1.2263602018356323, -0.4262333810329437, -0.06011354923248291, 0.01052609272301197, -0.003743050154298544, 0.4911783039569855, 0.0203951857984066, -1.9881715774536133, -0.47124966979026794, 0.07765527814626694, 0.007336748298257589, -0.23703432083129883, 0.013366902247071266, -0.7022212743759155, -0.5000659823417664, -0.0446147546172142, 1.6718616485595703, 0.14558593928813934, 2.0918498039245605, 0.11280177533626556, 0.1956515610218048, -0.12125277519226074, -0.011481503024697304, -0.020714327692985535, -1.5253804922103882, 0.17636367678642273, 0.09091463685035706, 1.0130555629730225, 0.21143265068531036, -0.6374700665473938, 2.7794225215911865, 0.10848954319953918, 0.3101067543029785, -0.4934631288051605, 0.48930850625038147, -0.07159571349620819, 0.19774284958839417, 0.30959734320640564, -0.0643114298582077, 0.8650744557380676, 0.008420308120548725, -0.15407031774520874, -0.17469552159309387, 1.0979814529418945, 0.2633243501186371, 1.8690377473831177, 2.664923667907715, 0.02069026045501232, 0.13936929404735565, -0.47460418939590454, -0.30848851799964905, -0.09351404011249542, 0.22160130739212036, 3.306893825531006, 0.16872534155845642, -0.801479697227478, -0.17013224959373474, -0.19784283638000488, -0.7567635774612427, -0.027192313224077225, -0.01646379753947258, -0.00567585788667202, 1.5336813926696777, -1.1752206087112427, 0.03553381934762001, 0.8292652368545532, -1.028513789176941, 0.773944079875946, 0.15032850205898285, 0.5277897119522095, -1.060196876525879, 0.6927959322929382, -1.971850037574768, -0.5849586129188538, -0.20490722358226776, 0.2549756169319153, 1.6778453588485718, -0.15137158334255219, -0.5919398665428162, 1.1280447244644165, 0.006428507622331381, -0.008641250431537628, 0.4132014214992523, 0.0080258809030056, 1.0815973281860352, 0.03156435117125511, -0.0001786612847354263, -0.06410878896713257, 0.522954523563385, -0.25710299611091614, 0.4066632091999054, 0.341462641954422, 0.07856407761573792, 0.033726394176483154, 0.14125210046768188, 1.0973283052444458, 0.2544885277748108, -3.766824722290039, 0.10962799191474915, -0.16065631806850433, 0.003411108162254095, 0.9209687113761902, -0.14408192038536072, 0.002613919088616967, -0.3170463740825653, -0.06454098224639893, 0.23116688430309296, 0.45021915435791016, 0.15041176974773407, 0.06276986747980118, -0.048638369888067245, 0.0045628962107002735, -0.06964634358882904, 0.08700749278068542, -2.0946919918060303, -0.2836054563522339, 0.2438337504863739, 0.03199802711606026, -0.11439678072929382, 0.25103071331977844, 0.38378989696502686, 0.4456547200679779, -1.218670129776001, -0.19883613288402557, -0.1523473709821701, -1.4485100507736206, -0.37610122561454773, 0.0981849655508995, 0.047241516411304474, -0.2589937448501587, -1.0830771923065186, -1.0253565311431885, -2.0815513134002686, -1.6915667057037354, -0.31737130880355835, 0.0825231596827507, -0.4098053574562073, 0.24324119091033936, 0.05755844712257385, 1.4977645874023438, 0.6030408143997192, -1.1437278985977173, 0.8722948431968689, -0.3202451169490814, 0.07511533051729202, -1.6856932640075684, -2.433535099029541, -0.26997116208076477, -0.1283850073814392, 0.21224172413349152, 0.9160288572311401, -3.5412254333496094, -0.6341166496276855, -0.5305447578430176, 0.1864151656627655, -1.5401484966278076, -0.4320002496242523, 0.042181346565485, 0.24172408878803253, 0.48234429955482483, -1.091723918914795, 0.13982291519641876, 0.7872896790504456, 0.9854079484939575, 0.362040638923645, 2.383561849594116, -0.798906147480011, 0.7152125835418701, 0.11264070868492126, -0.002483671298250556, 2.447319269180298, -0.09913337975740433, -0.2566579282283783, -0.3001483678817749, -0.63556307554245, -0.3697143495082855, -0.16264031827449799, -0.7545874118804932, -0.9607383608818054, -0.018444161862134933, 1.0506007671356201, 2.0842058658599854, 0.39943867921829224, 0.3233413100242615, 0.05585117265582085, 0.4655313789844513, -0.4455958604812622, -0.41527602076530457, -1.4916296005249023, 0.2410585582256317, 0.6797896027565002, 0.6645042300224304, 2.17191743850708, 0.4426341950893402, -0.26110565662384033, 0.5423992872238159, 2.889436721801758, -0.4987524449825287, -0.15290085971355438, 0.08365372568368912, -0.014922033064067364, 0.1868080198764801, 0.17015805840492249, -1.4984866380691528, 1.4428422451019287, -0.1482359915971756, 0.7350178956985474, 1.5592905282974243, -0.04210584983229637, -0.09345701336860657, 0.12383660674095154, -0.6742566823959351, 1.513379693031311, -3.1489336490631104, -0.06326925754547119, 0.9842807054519653, -0.16857604682445526, -0.976281464099884, -0.06356889009475708, -0.08299285918474197, -0.19188718497753143, -0.11747563630342484, 0.010889221914112568, 0.08632446080446243, 0.7490261197090149, -0.5979118347167969, -3.793114185333252, 0.5643714070320129, 0.4892268776893616, 0.12114787846803665, -2.350808620452881, 0.15284068882465363, 0.009472587145864964, -0.10225356370210648, 0.5866782069206238, -0.05500989034771919, 3.2446682453155518, 0.03581559658050537, 0.41961538791656494, 0.9602587819099426, 0.7441412806510925, -1.0583207607269287, -0.15557979047298431, -0.1861347258090973, 1.1682060956954956, -0.11432062089443207, 0.18954142928123474, -0.09302840381860733, 0.11665306240320206, 2.0441246032714844, -0.14420583844184875, 1.459697961807251, -0.25080233812332153, -0.35758939385414124, 0.7279442548751831, 0.11026699841022491, 1.8421133756637573, 3.316763162612915, -1.7347322702407837, 0.03782064467668533, -0.9230195879936218, 0.4996342062950134, 0.4270801544189453, -1.3742412328720093, -0.6554979085922241, 0.26204392313957214, 0.19155250489711761, 0.10635382682085037, -0.03653814271092415, -0.08955015987157822, -0.01578451320528984, -0.031458333134651184, -0.7049471139907837, 0.025317247956991196, 0.9543954730033875, 0.343910813331604, -0.0585249699652195, 1.5886814594268799, 0.49094459414482117, 0.0891362652182579, 0.24661141633987427, 0.019635504111647606, -0.45154112577438354, 2.419344425201416, 0.3538093566894531, 0.03464844077825546, 0.03689318522810936, -1.137276291847229, -0.2867608368396759, -0.35330405831336975, 0.13254785537719727, 0.20175161957740784, -0.07284455746412277, -0.37046846747398376, 0.01217273436486721, -0.07939811795949936, -0.035869378596544266, -0.30306369066238403, 1.5686671758885495e-05, -0.20506572723388672, 1.2708488702774048, 0.4769761860370636, -0.5788885951042175, 0.014856703579425812, 1.498033881187439, -1.827608585357666, 0.29178059101104736, -1.9396414756774902, 4.60159158706665, -0.44945117831230164, -1.6842106580734253, -2.162534475326538, 0.3327236473560333, 0.07484502345323563, -0.2983771860599518, 0.3153543472290039, -1.2896535396575928, -0.054295334964990616, -0.32726892828941345, 0.6176272630691528, -1.3240007162094116, 0.5734339356422424, 0.6509187817573547, 0.2782340943813324, 0.04511137679219246, -0.014120490290224552, -1.5282803773880005, 0.04613642022013664, 0.1522064059972763, -0.2426617592573166, 0.4922241270542145, -0.23164038360118866, 2.6729109287261963, 0.6285266876220703, -0.5167672634124756, 0.25463223457336426, 0.03568938747048378, 0.03830055892467499, -0.4735276699066162, -1.2303708791732788, -0.21781669557094574, -0.5983081459999084, 0.04337184503674507, 0.5219825506210327, 0.35400035977363586, 2.440638542175293, -3.896001100540161, 0.6033254265785217, 0.005453092511743307, -1.3231048583984375, 1.6266489028930664, 0.008082776330411434, -0.1869998723268509, 0.06620839238166809, 0.33376261591911316, -0.2301359921693802, 0.7575598359107971, 1.2691872119903564, 0.008436971344053745, -0.1937597095966339, 0.11383893340826035, -0.04222910478711128, -0.12556810677051544, 0.16828909516334534, -2.092365264892578, -2.5728747844696045, -0.024330176413059235, -0.4591502845287323, 0.966862678527832, 0.5686200261116028, 0.9356221556663513, -0.028878508135676384, 1.675266146659851, 0.7695584297180176, -0.6572775840759277, 0.44771113991737366, 2.800837278366089, -0.20450638234615326, -1.8218575716018677, 0.0020274855196475983, -0.16921599209308624, 0.8875271081924438, 0.9680615067481995, 0.4822559654712677, -1.1850864887237549, 0.6410632729530334, 0.7919716238975525, -1.6486639976501465, -0.003145656082779169, -0.15880896151065826, 1.1542515754699707, -0.8577254414558411, 2.6960089206695557, -0.07603272050619125, -0.3713184595108032, 0.07240511476993561, -0.33782026171684265, 1.920026421546936, -1.7351733446121216, 1.8241254091262817, -2.5589888095855713, 0.9278563857078552, 0.014479083009064198, 1.3838187456130981, 0.03691167011857033, -0.9022322297096252, 1.0551663637161255, -0.05406789481639862, 0.6547222137451172, -1.0718401670455933, -0.06897758692502975, -0.02110052853822708, -1.8982290029525757, -1.3471770286560059, 0.3334513008594513, -1.085262656211853, -0.2522698938846588, -1.903120756149292, 1.0787372589111328, -0.6044138073921204, -0.18411695957183838, -0.2581116855144501, -0.3980414569377899, 0.34355804324150085, -0.3236607313156128, 0.8455781936645508, 0.18400834500789642, -0.26899829506874084, 1.9034860134124756, 0.3352702856063843, -0.21417003870010376, 0.1405610293149948, 0.10233857482671738, 6.865728855133057, -0.3663436770439148, -1.6400290727615356, -0.1184893473982811, -0.12336220592260361, 0.9479421377182007, 0.13874764740467072, 0.004266653675585985, -0.07336829602718353, -0.4058081805706024, -0.011531037278473377, 0.02135406620800495, -0.432875394821167, -0.14180921018123627, 0.004505665507167578, -0.02648969180881977, 0.1810365915298462, -0.14376847445964813, -0.42028793692588806, -0.02877848409116268, 0.9226710796356201, -0.5238022208213806, -0.050444599241018295, -0.9970372915267944, 0.13468195497989655, -0.10108048468828201, 1.3892728090286255, -0.017219604924321175, -0.09330819547176361, -1.263335943222046, 0.538785994052887, 1.5355675220489502, -0.558918297290802, -0.2390180081129074, -0.1626758724451065, 1.4462391138076782, 0.49404776096343994, -0.4883621335029602, -0.18164946138858795, 0.014001177623867989, -0.594184160232544, -0.15119612216949463, 3.0223464965820312, -0.02183746173977852, 2.4160635471343994, -0.9233068823814392, 0.13750404119491577, 0.3705146014690399, 0.01508074440062046, 0.16642563045024872, 0.23632392287254333, -2.8416054248809814, -0.0571448877453804, 0.17990703880786896, -0.735292911529541, 0.022312460467219353, -0.5129411220550537, -0.1125253364443779, -0.2859457731246948, 0.04487758129835129, 0.0066828420385718346, 0.042449578642845154, -0.7291842103004456, -0.2115887850522995, 1.2144583463668823, 0.4228280782699585, -0.35106563568115234, 0.35988619923591614, -0.3372129797935486, 1.1341705322265625, 0.2287246435880661, 1.8542441129684448, 3.0736351013183594, 0.7849892973899841, -0.575376570224762, -3.618485689163208, 2.569645881652832, -0.3237505853176117, -0.207097589969635, -0.5916334390640259, -0.08229783922433853, -0.6654956936836243, -0.05972391366958618, 0.1747625470161438, 0.3350599706172943, -1.898535966873169, -0.0026675118133425713, -0.14048099517822266, 0.01059829629957676, -0.27567189931869507, -0.12416322529315948, 2.3948569297790527, 1.3831729888916016, 0.22061368823051453, -0.412557989358902, -0.5411081314086914, -1.6504998207092285, 0.12990707159042358, 0.007552432827651501, -1.011920690536499, 2.6490871906280518, -0.15758001804351807, -1.3717341423034668, -0.10039827227592468, 0.05123355984687805, -1.8878742456436157, 0.3379952311515808, 0.3360494375228882, 0.9734981656074524, -0.030122095718979836, -0.0930224359035492, -1.7817484140396118, -0.014386450871825218, -0.2781178057193756, 0.047505151480436325, -0.049245432019233704, -1.4264750480651855, -0.13335411250591278, 0.13242274522781372, -0.6860639452934265, -0.5124555826187134, 0.09090910851955414, 0.033091720193624496, -0.6042055487632751, -0.2829922139644623, 0.9546623229980469, 0.02264738269150257, 0.30362269282341003, -0.012959475629031658, 0.14685770869255066, -0.49478551745414734, 0.06319642812013626, 0.6907291412353516, 0.1300920695066452, 1.3277240991592407, -0.1494932323694229, 0.10366789251565933, 0.3215999901294708, -0.7820353507995605, -1.1434886455535889, 2.2519450187683105, -3.30776309967041, 0.23668122291564941, 0.04243597388267517, 3.2676961421966553, -0.016893252730369568, 0.060391440987586975, 0.12090163677930832, -0.16205357015132904, 1.1377687454223633, -0.4142364263534546, 0.07395763695240021, -0.05039186775684357, 0.660283625125885, -2.7939085960388184, -1.2891465425491333, -1.6400700807571411, -0.4297468066215515, -0.10368947684764862, -0.3353051245212555, -0.3891899287700653, -0.8961806893348694, -0.4913669228553772, -0.07943587005138397, 4.808444499969482, -5.729074478149414, 0.05485839024186134, 0.8266882300376892, 0.23988454043865204, -0.05589508265256882, 0.4954429864883423, 2.0502796173095703, 0.09579421579837799, -0.07104693353176117, 2.5147385597229004, -0.06903699785470963, -0.21391816437244415, 0.08129631727933884, -1.952298879623413, -0.4286453425884247, 0.17580682039260864, -0.24009618163108826, -0.32854408025741577, 0.3290155231952667, -1.1639900207519531, -0.4733704626560211, -2.716256618499756, -0.07172402739524841, 1.7371923923492432, 1.2711093425750732, -0.7004410028457642, -0.346235454082489, -0.11790431290864944, 1.3327155113220215, 0.7396570444107056, -0.8365303874015808, 0.5213953852653503, -0.25477296113967896, -0.7785399556159973, 0.012662715278565884, -0.16675862669944763, -1.4011366367340088, 0.06612391769886017, -0.6298055052757263, 0.02532181143760681, 0.4982667863368988, 0.0033958766143769026, -0.006691065616905689, 1.3874759674072266, -0.4504584074020386, 0.07299631088972092, 0.43551528453826904, -0.0014330875128507614, -0.052766021341085434, 1.0582658052444458, 0.5345312356948853, 1.0780085325241089, -2.355724811553955, 1.4195225238800049, -1.032699704170227, -1.483984351158142, -0.10116500407457352, 0.11688262224197388, 0.05589523911476135, -0.23061315715312958, 0.053747229278087616, 1.5548714399337769, -0.0007365670171566308, -0.0043976204469799995, -0.2803318500518799, -0.5807315111160278, 2.512817621231079, -0.3669123947620392, -0.3823828399181366, -0.05153176933526993, 0.34022781252861023, -0.00909759383648634, 1.0627880096435547, 1.025197982788086, 0.0411120168864727, 0.004164241719990969, 0.03657493367791176, -1.1590213775634766, -0.1345599889755249, -0.32044893503189087, -3.52352237701416, 0.014582748524844646, -0.34927791357040405, -0.0024327225983142853, -0.1211700364947319, -0.3650388717651367, 0.013873969204723835, 2.5264580249786377, 0.49597540497779846, -1.4276797771453857, -0.20134824514389038, -1.3763993978500366, -0.22309820353984833, -0.059426672756671906, 1.0654854774475098, 0.005056455731391907, -0.0019080104539170861, 0.02646726369857788, -0.023141920566558838, -0.47390812635421753, 0.1397065371274948, 1.7761677503585815, 0.010789196006953716, -0.4092537462711334, -0.25052088499069214, 1.6746772527694702, 0.0934314951300621, -0.46865710616111755, -1.1708120107650757, 0.18540769815444946, 0.866125762462616, -1.3644031286239624, -0.02742355689406395, 2.0831711292266846, -0.021408136934041977, 0.18508197367191315, 0.013007649220526218, 0.1292302906513214, 0.06176619604229927, -0.6279188990592957, 0.05364887788891792, -0.9132556319236755, 0.1925867348909378, -2.0078275203704834, -0.3519115746021271, -2.1767494678497314, -0.16179384291172028, 0.008383381180465221, -2.6411733627319336, 0.6607449650764465, -0.17833960056304932, 0.7949861288070679, -0.2790400981903076, -0.23324847221374512, 0.12554039061069489, -0.7303617000579834, 0.5941137671470642, 0.8576824069023132, 3.1374216079711914, 0.9879631400108337, -0.949741005897522, -0.11796490848064423, -0.11246111989021301, 0.18237943947315216, -0.5595958232879639, -0.24895651638507843, -0.5227086544036865, 0.7636558413505554, -0.8414403796195984, 0.0006787730380892754, -0.16742287576198578, 0.1570039689540863, 0.5947840809822083, -0.1986730545759201, -0.08944904804229736, 0.3474695086479187, -1.7587038278579712, 0.09206719696521759, 0.42531102895736694, 8.042403351282701e-06, -0.3384912312030792, 0.14176979660987854, -0.37097683548927307, 1.5994161367416382, -0.011541968211531639, -0.19386641681194305, 0.09314993768930435, -0.555448055267334, 0.10478434711694717, -0.33407631516456604, 0.5576615929603577, 0.04934743791818619, 0.774093747138977, 0.24926239252090454, -0.0038157631643116474, 0.36579108238220215, 0.8377469778060913, 0.0026288439985364676, -1.096372365951538, -0.0038253157399594784, 0.30156052112579346, -0.38627132773399353, 0.9996780157089233, 0.9317198395729065, 0.8017498254776001, -0.1916871815919876, -0.09232881665229797, 0.03414975851774216, 0.03897789493203163, -0.5405380725860596, -0.5858137607574463, 0.1559189409017563, 0.8991641402244568, 0.03179891034960747, -0.6476821303367615, 0.5745470523834229, 1.0401910543441772, -1.7920489311218262, -0.8771219253540039, -0.6940984725952148, -1.15311598777771, -0.24366123974323273, 0.13547754287719727, -0.9671984314918518, -0.005548177752643824, -0.5779871940612793, -0.0571632981300354, 0.18688717484474182, -0.07089771330356598, -0.040686286985874176, 0.1513795703649521, 0.6562063097953796, -0.9397801756858826, 0.7575087547302246, 0.12229613214731216, 0.6214740872383118, -0.44801411032676697, -0.38344141840934753, 0.5144438147544861, -0.4358710050582886, -1.5161194801330566, -0.027509918436408043, 0.1030731126666069, -1.1660054922103882, -0.28321486711502075, 0.5557054877281189, -0.20048929750919342, 0.02063898928463459, 1.3710062503814697, 0.5255920886993408, 0.20624923706054688, 0.11135261505842209, -1.014365792274475, 0.17290747165679932, 1.603798747062683, -0.17344829440116882, 0.9175575375556946, 0.11222327500581741, -3.4883241653442383, -0.159200519323349, -1.2418750524520874, -0.8061889410018921, -0.3900170624256134, 0.30573466420173645, -0.47241994738578796, 0.09019548445940018, -0.0064682066440582275, 0.006855478510260582, 1.422196388244629, 0.5390423536300659, -2.999680280685425, -0.1872132271528244, -0.0657777339220047, -0.472486674785614, -0.11605514585971832, 0.1060311570763588, 0.033643994480371475, 0.24788227677345276, -2.0717673301696777, -0.6273764967918396, 0.07022860646247864, 0.3938967287540436, -1.4860317707061768, 2.5464298725128174, 0.5624433159828186, -2.4489521980285645, -0.8550205826759338, -1.9591177701950073, 0.0018959793960675597, -0.19744454324245453, -0.5343562960624695, 1.519140601158142, -3.157783269882202, 1.3617860078811646, 0.005571397487074137, -0.9287988543510437, -0.08066624402999878, 0.2782559096813202, -0.3834848701953888, 0.44693297147750854, 0.06822386384010315, -0.532809317111969, -0.6675519943237305, -0.424723356962204, 0.06342015415430069, -0.5656905770301819, 0.3511754870414734, -1.2285875082015991, -0.011573022231459618, -0.0026255021803081036, -0.6168316602706909, 0.40418052673339844, -2.372309684753418, -0.18444481492042542, 0.3261093497276306, 0.004337828606367111, -0.7865056395530701, -1.3926475048065186, 1.6949735879898071, -0.07052979618310928, -0.9517403244972229, 1.1372812986373901, -2.0666675567626953, -0.49036794900894165, 0.6429658532142639, -0.15880261361598969, 1.6181352138519287, 0.07612620294094086, -1.862828254699707, -0.2839924395084381, 0.2562878727912903, 0.31702035665512085, 1.0548774003982544, -0.9881041049957275, -1.9085315465927124, -0.362214595079422, -1.6940929889678955, 0.16389137506484985, 0.006723838858306408, -0.8399493098258972, 0.06607118993997574, -0.7861194014549255, 0.1363368034362793, -0.3424990475177765, -0.002282925881445408, -0.1620076596736908, 0.09526974707841873, 1.7594349384307861, 0.08961965888738632, 0.1437160223722458, -0.010607805103063583, -1.1182292699813843, -0.5222141742706299, -0.28741979598999023, -0.5940712094306946, 1.5059149265289307, -1.110613465309143, -0.2693558633327484, 0.8123021721839905, -0.7316348552703857, -0.0034170050639659166, 0.24586093425750732, 0.21295307576656342, 0.01958056166768074, -0.08039811998605728, 0.19436508417129517, -0.6878920793533325, -0.22618629038333893, 0.07677113264799118, -1.16084623336792, 1.0664986371994019, 0.8879098296165466, -1.0830780267715454, 0.4794205129146576, -1.4638378620147705, 0.1298372447490692, -1.478027105331421, 0.1963483840227127, -3.8560104370117188, -0.5456003546714783, -0.322394996881485, -2.9725899696350098, -1.1051652431488037, 0.9801626205444336, 0.736721932888031, 0.23651733994483948, 0.17536874115467072, -0.08260077983140945, -0.29750198125839233, -0.5204691886901855, -0.0015128037193790078, 0.8835853338241577, -0.8172277212142944, 0.11553287506103516, 6.1200361251831055, 0.5903319716453552, 0.01414831168949604, 5.201208114624023, -0.45768022537231445, 0.9327218532562256, -0.026164010167121887, -1.5486153364181519, -0.6403862237930298, 0.029438327997922897, -0.1539524495601654, -0.07592958211898804, 0.1606738120317459, -1.7599568367004395, -0.015309050679206848, -0.19158321619033813, -0.6095903515815735, -0.3136886656284332, -0.7905071973800659, -0.1758916974067688, 0.710793137550354, 0.00379493972286582, 1.0934703350067139, -1.022367000579834, -0.39910396933555603, -0.2295013666152954, 0.04979440197348595, -0.3724033534526825, 0.1972808539867401, -0.07195694744586945, 0.11093638837337494, -0.10815411061048508, -1.6277215480804443, -0.48087865114212036, -0.16840432584285736, -1.106479525566101, -1.418479323387146, 0.15416036546230316, 0.035471782088279724, -0.1780424863100052, 0.7988128662109375, 2.6623895168304443, 0.36811351776123047, 0.01858600787818432, -0.1035265401005745, 0.7609795331954956, 0.1176694855093956, 0.14759083092212677, -0.29642510414123535, -0.01817004755139351, -0.4615805447101593, -0.6236655116081238, -1.2975099086761475, -0.18082422018051147, 0.30826491117477417, 1.9125144481658936, -0.6257483959197998, -0.17167581617832184, -0.002750539453700185, -0.18002961575984955, 0.01638833060860634, -0.18312008678913116, -0.6198801398277283, -0.2883022129535675, -0.28361812233924866, 0.15433116257190704, 0.0346643328666687, 0.14330707490444183, -0.8780875205993652, -0.005705888383090496, -0.13091270625591278, -0.6169025897979736, 0.29962417483329773, -0.02197815291583538, 0.1386299878358841, 0.13680633902549744, -0.2467910200357437, -0.5230150818824768, 0.24158184230327606, 0.03785914182662964, -0.02778244949877262, -0.4865292012691498, -0.9954386949539185, 0.934137761592865, 0.4212324023246765, -0.2681507170200348, 0.5124803185462952, -0.015587503090500832, 3.2614214420318604, -2.249990940093994, -0.12129279226064682, -0.7739551067352295, -1.0167068243026733, 0.06188403442502022, 0.3747565448284149, -0.4185125231742859, -0.03902805969119072, 0.20801828801631927, -0.26709771156311035, 0.8101481795310974, -0.03617649897933006, -0.5350621938705444, -0.023049328476190567, -0.2930172383785248, -0.6015669703483582, 0.4575231671333313, -0.14343412220478058, -0.5547783374786377, -0.35115841031074524, -0.1516181081533432, -0.5245852470397949, -0.06263434886932373, -1.1233848333358765, 0.37669169902801514, 2.9456779956817627, -0.004832169506698847, 0.000302140018902719, -0.05110103264451027, -0.22055913507938385, 2.8601560592651367, -0.01998363994061947, -0.13364754617214203, 0.37690815329551697, 0.36084556579589844, 1.09213125705719, 0.7569886445999146, 0.32698187232017517, 0.2291296124458313, -2.042585611343384, -0.5958592295646667, -0.13414950668811798, 0.2598823606967926, 0.1032329723238945, -0.08890955150127411, 0.01829858310520649, 0.15105776488780975, 0.03858548030257225, -0.08331223577260971, -0.5372778177261353, 2.223576307296753, 0.26823651790618896, -0.2867506444454193, -2.8591628074645996, -0.4967215061187744, -0.24221068620681763, -0.28625035285949707, 0.8986853361129761, -0.04343276098370552, -0.9614626169204712, 0.15763606131076813, 0.3838079571723938, -0.2774093449115753, -1.1423910856246948, -0.08613195270299911, 0.16950836777687073, -0.2531576454639435, 0.09978679567575455, -0.12962186336517334, -0.6270723938941956, 0.9601505994796753, -1.359397530555725, 0.04571959003806114, 1.9726901054382324, 0.1717202365398407, -0.11535292118787766, -0.521690309047699, 0.6506881713867188, -0.5283432602882385, 0.2302447259426117, -0.32288888096809387, -0.21697022020816803, 1.197587251663208, 0.014845890924334526, -0.36180204153060913, -0.16253559291362762, -1.1881766319274902, 0.268928200006485, 0.4169856011867523, 0.4533391296863556, 0.46433529257774353, 0.1531156450510025, 0.1368606835603714, -0.14133776724338531, -0.6072680354118347, -0.30036255717277527, -0.30532750487327576, -0.03586681932210922, 0.0016257289098575711, -0.35606351494789124, 1.040884017944336, -0.5861097574234009, 1.327386736869812, 1.1464937925338745, 1.7332683801651, -1.349377989768982, -0.3115155100822449, 0.016643518581986427, -0.05464564636349678, -2.28794264793396, 0.08741308003664017, 0.056343793869018555, 0.08431359380483627, 0.789348840713501, -0.9837659001350403, 0.02736930549144745, 3.9781606197357178, -2.1431796550750732, -2.857069492340088, -1.4963362216949463, -0.008058356121182442, -0.21058306097984314, 0.014749091118574142, 0.9391854405403137, 0.2010369896888733, 0.373575896024704, 0.06922531872987747, -0.08365300297737122, -0.0787699967622757, 0.1510411500930786, -2.1962316036224365, 0.4628428816795349, 0.000310256757074967, 0.6672167181968689, 0.5793853402137756, -0.1570202261209488, -1.2908343076705933, -0.2943928837776184, -0.3620385527610779, -0.8060998916625977, 0.06001662835478783, -0.36308810114860535, 0.1453811228275299, -0.16249094903469086, 0.36098721623420715, 0.07197174429893494, -0.9374801516532898, -0.5405752658843994, 0.17377948760986328, -0.23324456810951233, -0.04926604405045509, 3.480538845062256, -0.0336310900747776, -0.06967317312955856, 0.022021779790520668, 0.026555703952908516, 1.1604384183883667, 0.755741536617279, -0.42609187960624695, -0.1252894252538681, -1.404009222984314, -0.8606547117233276, -1.7830164432525635, -0.43749818205833435, -0.056886207312345505, 0.7594017386436462, 0.5648028254508972, 0.14395971596240997, -0.03179623931646347, -0.12479189038276672, 0.04709366708993912, 1.0865312814712524, 0.3225899040699005, -0.26027438044548035, -0.10882121324539185, -0.5374701023101807, 0.357835054397583, 0.7299017310142517, -0.0033634405117481947, 0.19433419406414032, 0.9091987013816833, -0.03281653672456741, -0.10562275350093842, -0.29013675451278687, -0.7698548436164856, 1.3094935417175293, 1.5112144947052002, 0.09730826318264008, -0.10870611667633057, -0.1080261841416359, 0.22826386988162994, 0.3757932484149933, 0.5062510967254639, 0.3426226079463959, -1.8424471616744995, -0.08172667771577835, -2.483466625213623, 0.008494116365909576, 1.0097846984863281, 0.42629274725914, 0.0664738118648529, 1.6506829261779785, 0.5078133344650269, -2.770949125289917, -0.15406905114650726, 1.3709574937820435, 0.37260812520980835, 0.02909359708428383, -0.7286941409111023, 0.12860600650310516, -2.2520015239715576, 0.006686845328658819, 0.10430368781089783, 0.6448007822036743, -0.24872055649757385, -0.5252789855003357, 1.5956534147262573, 0.019781148061156273, -0.09624634683132172, -1.2104196548461914, -0.005839561112225056, -0.15212960541248322, -0.4149321913719177, 0.4276125133037567, -0.2246163934469223, -0.041451595723629, 0.7655320763587952, 2.0270047187805176, -0.3233395218849182, 0.44574323296546936, -0.024437742307782173, 0.04584415629506111, 2.0570249557495117, -0.8462827801704407, 0.06456256657838821, 0.024722963571548462, 0.06732750684022903, -0.03865128383040428, -0.006745075806975365, -0.04084407910704613, -2.828477621078491, 0.3293863534927368, -0.14825695753097534, 0.13144361972808838, -1.5629802942276, -1.051722764968872, 0.5200347900390625, 0.10813973844051361, 2.205409049987793, -0.8786084651947021, 0.42920932173728943, 0.010067258030176163, -0.38643327355384827, 0.9203000068664551, -0.31000977754592896, 0.2766563296318054, -0.7743930220603943, -1.273736596107483, 0.22329752147197723, -0.23860615491867065, -0.13778848946094513, -0.24987386167049408, -0.8702234625816345, -1.5593146085739136, -1.0949897766113281, -0.0760476142168045, 1.077815294265747, -0.00013193771883379668, -0.4435494542121887, -0.42821231484413147, -0.32940253615379333, -0.6279798746109009, 0.8618227243423462, 0.39397484064102173, -0.34118884801864624, 0.6578445434570312, 0.15712888538837433, 1.1131120920181274, 0.5188966989517212, 0.7949565649032593, 1.612216591835022, -3.3845386505126953, -0.5406860113143921, -2.0458602905273438, -0.5156828761100769, -1.5292936563491821, 1.8356040716171265, 1.4735757112503052, 0.676891565322876, 0.042873844504356384, -3.113363742828369, 0.7942565679550171, -1.4433753490447998, -0.11183926463127136, 0.2486751526594162, -0.6165104508399963, 0.8456897139549255, 0.0031614273320883512, -0.16271494328975677, 0.036339566111564636, -1.3494160175323486, 0.2422909140586853, 0.0038319106679409742, -0.07934504747390747, 1.4408682584762573, -0.07629261165857315, -2.204213857650757, 3.3866546154022217, -0.9080508947372437, -0.009781730361282825, -0.2565501034259796, 0.4613248407840729, 0.998304545879364, 1.3322707414627075, 0.09957516193389893, -0.4233473837375641, 0.20310133695602417, -1.5481946468353271, -0.6613205075263977, 0.04602232202887535, -0.22337082028388977, 0.14034011960029602, 0.1531558483839035, 0.6303767561912537, 0.7625194787979126, 0.0604233480989933, 0.8451488018035889, -0.001766573404893279, 0.07374967634677887, -2.1700940132141113, -0.5257042050361633, 0.22058212757110596, -0.22346210479736328, 0.4443206191062927, 0.09400927275419235, -0.1143423393368721, -0.5421828031539917, -0.33154812455177307, -0.22633671760559082, 0.03860710561275482, 0.3073822855949402, -1.300083041191101, -0.9915681481361389, -0.05812878534197807, -0.17104367911815643, -0.8958588242530823, 0.015191015787422657, 0.3873036503791809, 0.45157256722450256, -0.2600325644016266, -0.6787373423576355, -0.5268266797065735, 0.27449214458465576, -0.06046316400170326, 0.014532273635268211, 0.3809299170970917, 0.18769356608390808, -0.020806869491934776, -0.20809167623519897, 0.23893137276172638, -0.06901361793279648, 0.8617782592773438, 0.3107842803001404, 0.035477928817272186, 0.047413840889930725, -1.9914014339447021, -0.23655559122562408, 0.654516875743866, -0.22030068933963776, -0.5178086161613464, -0.20911742746829987, -0.44223806262016296, 0.11948616802692413, 0.29187220335006714, -0.0364263579249382, 0.8358064889907837, -0.11291435360908508, 1.1816359758377075, -3.7012417316436768, 0.8119268417358398, 1.1115397214889526, 0.9701239466667175, 0.13240037858486176, 0.8056516051292419, -0.41606372594833374, 0.19398775696754456, -0.2508120834827423, 0.6062778234481812, 0.5833867192268372, -0.4422183334827423, -0.9779468178749084, -0.3592323958873749, 0.4191734492778778, -0.007257153280079365, 0.9495880007743835, -0.02941003628075123, 0.13803613185882568, 0.4375985860824585, -0.3159502446651459, 0.3287625014781952, 0.11009462177753448, 0.3814660608768463, 0.4716801643371582, 0.43642961978912354, -1.0906658172607422, -0.1833944320678711, 0.3570702075958252, 0.3786362111568451, -0.12740816175937653, 0.17567890882492065, 0.019769050180912018, 0.9166865348815918, 0.9032813906669617, -0.16226764023303986, 2.3619678020477295, 0.4335118234157562, 2.924412727355957, 0.04750113934278488, -0.07486989349126816, 1.6260733604431152, -0.11282748728990555, 0.020612485706806183, -0.9252427220344543, -0.2299066036939621, 2.775906801223755, 0.5234481692314148, -0.005341924726963043, 0.5519295930862427, 0.07734819501638412, -1.0891799926757812, 0.010937959887087345, -1.3375134468078613, 0.22688227891921997, -0.11644116789102554, -0.7713035345077515, -0.182670459151268, -0.9295128583908081, 3.4786341190338135, 0.20517845451831818, 1.4863157272338867, 1.8672021627426147, 1.0095795392990112, -0.806122899055481, 0.42092978954315186, 3.4046144485473633, -1.4634202718734741, -2.346369743347168, 0.0018300508381798863, -0.052211470901966095, 0.11871709674596786, -0.1484154760837555, -0.11421509832143784, -0.11213406175374985, -1.5272623300552368, 2.467024564743042, -0.680162250995636, -0.09279969334602356, 0.01301902998238802, 0.2804237902164459, -0.5148439407348633, 0.4792337715625763, -1.9603105783462524, -2.3229422569274902, 0.05020398274064064, 1.5942330360412598, -0.12871044874191284, 0.5962604284286499, -0.6285696625709534, 0.06645771861076355, 1.2475523948669434, -0.6417968273162842, 0.39741483330726624, 1.3244394063949585, 2.760385036468506, -0.561205267906189, -0.2086862325668335, 0.9899629950523376, -0.3590563237667084, 0.11368773877620697, -0.26338815689086914, 0.1629626452922821, -1.238963007926941, 0.6502904891967773, -0.9053144454956055, 1.7024189233779907, -0.2559519112110138, 0.10774466395378113, -0.31107422709465027, -1.9453808069229126, -0.17166799306869507, -1.082281231880188, -0.05013163387775421, -0.15968453884124756, 0.6714406609535217, -0.3765416145324707, -0.016159189864993095, -0.4218411147594452, 0.08420366048812866, -0.16907113790512085, -0.8475294709205627, 0.0035126882139593363, 0.4590803384780884, 0.0615457147359848, 0.9979257583618164, -0.024261951446533203, -3.252237558364868, -0.47061291337013245, 1.0482538938522339, -0.7029632925987244, 0.4270665645599365, 0.8685235381126404, -0.022410938516259193, -0.040167611092329025, 1.929916501045227, 0.06709235906600952, -1.7572687864303589, -0.06862127035856247, -0.03410486876964569, -0.04445985332131386, 0.46248435974121094, -1.6497377157211304, -0.01992432028055191, 0.18590104579925537, -0.08203290402889252, 0.5072835683822632, 0.16191361844539642, 0.1325417160987854, 0.11725228279829025, -0.9147279262542725, 0.36495012044906616, -0.9081978797912598, 0.16255725920200348, 1.0257409811019897, 1.016571044921875, 1.02791428565979, -0.2074815183877945, -0.48081642389297485, 2.0089643001556396, -1.8575254678726196, -0.011372340843081474, -0.41146397590637207, 0.12034611403942108, 0.46949681639671326, -0.20922210812568665, -0.44166237115859985, 0.14330288767814636, -0.5032984614372253, 1.9580988883972168, 0.3283042013645172, -0.23355194926261902, 0.2491699457168579, -0.6084193587303162, 0.14956246316432953, 0.14942191541194916, 0.31489354372024536, 1.3022103309631348, 2.187861919403076, 0.03271671384572983, 0.6587621569633484, -1.3603910207748413, -0.7589756846427917, 0.4238438010215759, 0.2981950640678406, -0.8825951814651489, 0.1592121571302414, 0.11535583436489105, 0.2651103734970093, -0.869347870349884, -0.26691585779190063, -0.5965648889541626, 0.17772285640239716, 0.7340824604034424, 0.033184658735990524, -0.4227388799190521, 0.3576243221759796, 0.41986334323883057, -0.1794390082359314, -0.9466641545295715, -0.12774726748466492, 0.3707849383354187, -1.9704777002334595, 0.2669265568256378, -0.7947134375572205, -0.5916236639022827, 0.28184106945991516, 1.7901276350021362, -0.34649524092674255, -0.18505153059959412, 0.2737983167171478, -0.0005386436241678894, -0.9068077802658081, -0.02993754670023918, 0.10144682973623276, -0.001977668609470129, -1.4614272117614746, 0.7545378804206848, -0.013941753655672073, -0.11639294028282166, 0.24529585242271423, 0.055276259779930115, -0.23229162395000458, -0.45407888293266296, -0.020637579262256622, -0.4846377372741699, -0.640055775642395, -0.2255469113588333, 0.06961514055728912, -0.6571597456932068, -0.06584776192903519, 0.14847645163536072, -0.5769447684288025, -0.09147433936595917, -0.01078064739704132, -0.6740275621414185, 0.10323134064674377, -0.9823901653289795, -0.2594558596611023, -0.17408433556556702, 1.4932852983474731, 0.153437077999115, 2.0698320865631104, -1.343345046043396, -0.6534767150878906, -0.16608935594558716, 0.02336255833506584, 0.15284843742847443, -1.5091331005096436, -0.7332254648208618, 1.629159927368164, -0.31925663352012634, 0.05561651661992073, 0.06913137435913086, 0.10007110983133316, -0.07226750254631042, 2.2130393981933594, 1.2448410987854004, 0.12823516130447388, -0.27579808235168457, 1.8878583908081055, 0.001473233220167458, 0.3897189199924469, -0.10465344786643982, 1.1091349124908447, -0.10428839921951294, -0.3867104649543762, -0.15647494792938232, -0.03546459600329399, 0.15792953968048096, 0.40043389797210693, 2.4659478664398193, -0.0034118981566280127, -0.019624268636107445, 0.1231418028473854, 0.013979450799524784, -0.16875484585762024, 0.27282389998435974, -0.031306054443120956, 0.06918206810951233, -0.4016861915588379, -0.07005029916763306, 0.6684262752532959, -0.018435750156641006, 0.2909306287765503, -0.25799378752708435, -0.18664664030075073, 0.7026487588882446, 0.23549412190914154, -1.666900873184204, 0.4007246196269989, 0.8052436709403992, 0.061660703271627426, -0.9303416609764099, 0.4786682724952698, -0.598015546798706, 0.4581480324268341, -0.03549177944660187, -0.8410828709602356, -0.11549648642539978, 2.28377366065979, -0.7683191299438477, 0.007398100104182959, -0.05925876274704933, -0.33563029766082764, -0.026605110615491867, 0.1724855601787567, 0.3416479825973511, -0.4390181303024292, 0.9858946800231934, -1.3616726398468018, -0.467871755361557, -3.0613338947296143, -1.6456388235092163, -0.4264194369316101, -0.45487457513809204, -0.07423228025436401, 0.44433265924453735, -1.2034810781478882, -0.5901461243629456, 0.009872952476143837, 0.7039356827735901, -0.29824283719062805, 1.6847498416900635, 0.01175845880061388, 0.25210511684417725, 0.03896715119481087, 0.6162576675415039, -0.08965833485126495, -0.5072168111801147, 0.08142977952957153, -0.3558444380760193, -1.6114320755004883, 1.2322450876235962, -0.03582008183002472, -0.08365701138973236, 0.870216965675354, -0.07409803569316864, -0.00427462812513113, -0.2881333827972412, 0.023852886632084846, 0.06260834634304047, -0.20940935611724854, 1.169432520866394, 0.03241636976599693, -0.6347405314445496, -0.07872394472360611, 0.1559581756591797, -2.374979019165039, -1.083428978919983, 0.4021027088165283, -0.7175694704055786, -0.6158148050308228, -2.4772157669067383, -0.45712968707084656, 0.010435735806822777, 0.33023756742477417, 0.2684818208217621, -2.4944536685943604, -0.15574060380458832, 0.14598818123340607, 1.0979907512664795, 1.2167457342147827, 0.10292554646730423, 0.3087049126625061, 0.001667349017225206, -0.2848527729511261, 0.5975919961929321, 0.20814764499664307, -2.1252644062042236, -0.2473340928554535, 0.07420051842927933, 0.09216184914112091, 1.3943878412246704, 2.059664249420166, -0.22325275838375092, 0.10749441385269165, 0.18203894793987274, 0.060888245701789856, -2.9587085247039795, -0.6741801500320435, 0.8058516383171082, -1.09565007686615, -0.036988530308008194, -1.0515639781951904, 0.8106173872947693, 0.4514903128147125, 1.5937806367874146, 0.199277862906456, 0.17721521854400635, 0.19219546020030975, -0.4788687229156494, 3.4958529472351074, 0.151017427444458, -0.815098226070404, 0.09765954315662384, 0.3607478141784668, 0.3449050486087799, 0.08351898938417435, -0.0972624197602272, 0.04616709426045418, 0.7288211584091187, 0.4353870749473572, 0.4967043697834015, 0.5325488448143005, 0.5259260535240173, -0.7394488453865051, 0.516822874546051, -0.3893268406391144, 0.006208912935107946, 0.16924551129341125, 0.22548523545265198, -0.03175244480371475, 1.4292763471603394, 2.1219494342803955, -0.5887822508811951, -0.895786464214325, -0.2630876898765564, -0.3741266429424286, 0.7204797863960266, -1.1461334228515625, -2.794635057449341, -0.07634089887142181, -0.009608999826014042, 0.032637715339660645, 0.013861473649740219, -0.12431308627128601, 3.1501946449279785, -0.05698392167687416, -0.7044796943664551, 0.16191500425338745, -0.8899958729743958, 0.6102209687232971, 1.7468503713607788, 0.5356589555740356, 0.007255139295011759, -0.24354413151741028, -0.10338948667049408, 1.519781231880188, -0.022894030436873436, 1.0519987344741821, 0.9532312154769897, -0.577848494052887, -0.9357854127883911, -0.36440402269363403, -0.9435424208641052, 0.02822628803551197, 0.5627121329307556, -1.2252060174942017, -0.3525955080986023, -2.0143065452575684, 0.3790317177772522, -0.3667551875114441, 0.37129703164100647, -0.2070060521364212, 0.5453715920448303, -0.8250361680984497, 0.20604614913463593, 0.4917279779911041, -0.6298667192459106, 0.05435679852962494, 2.3158111572265625, 0.2983554005622864, 0.2032068520784378, -1.1077841520309448, -0.6686075329780579, 0.005962201859802008, -0.4103641211986542, 0.21170322597026825, -0.2111501842737198, -0.6637662649154663, -0.7870666980743408, -0.09711850434541702, 0.001329472055658698, 1.6550085544586182, 0.45420199632644653, -0.030404362827539444, 0.8630784749984741, -0.5633813142776489, 0.023295128718018532, 0.01492123119533062, -0.2021244466304779, 0.07296270877122879, -1.320910930633545, -0.029896438121795654, 0.25091713666915894, 1.7743111848831177, 0.6189221739768982, 0.2177029401063919, 1.1723356246948242, -0.044958584010601044, 1.0190412998199463, -0.2287013679742813, -0.17445072531700134, -0.2575684189796448, 2.347501277923584, 0.21397665143013, 0.1862158328294754, -0.46470463275909424, 0.6430662274360657, 0.674721896648407, 0.38224947452545166, 0.056990914046764374, -0.020398858934640884, 1.5927313566207886, 0.5749981999397278, -0.08665991574525833, 0.10808945447206497, -0.21874374151229858, 0.1706588864326477, 2.2382214069366455, 0.09003613144159317, -0.6578563451766968, -0.08502338081598282, -8.133396477205679e-05, -0.10287680476903915, -0.2866747975349426, -0.8878961801528931, -0.9385719299316406, 0.26014596223831177, -1.4856884479522705, 0.3500033915042877, 0.7186461091041565, -0.1433708518743515, 0.10788970440626144, 1.148198127746582, 0.6624624729156494, 0.3349388539791107, 0.06048234552145004, -0.004201511852443218, 4.758389949798584, 0.4779379367828369, -0.7713534235954285, -0.6883564591407776, 0.9873771667480469, -0.7329003214836121, -0.2483620047569275, -0.026607953011989594, 0.12744967639446259, 0.15868805348873138, -0.020025860518217087, -0.6609698534011841, 0.04586284980177879, 0.36028793454170227, 0.3253770172595978, -0.3155965805053711, 0.6459602117538452, -0.25384390354156494, -0.050395164638757706, 0.9329275488853455, -0.008809426799416542, 0.03503796085715294, -0.5299922227859497, -0.34820693731307983, 1.0640106201171875, 0.6526580452919006, 0.1333436518907547, 1.76070237159729, -0.00017700614989735186, 0.3708752989768982, 0.09952732175588608, -1.3538113832473755, -0.02042662724852562, 0.917391300201416, 0.19482077658176422, -0.11707515269517899, -0.0004317084967624396, 1.2139431238174438, -0.03400692343711853, -0.02183515764772892, -0.07784473150968552, -2.092494487762451, -0.4255179166793823, -0.25832125544548035, 0.253937304019928, 0.2214515656232834, -0.1068217009305954, 0.3976811170578003, -0.2570495307445526, -1.0105726718902588, -0.02436935156583786, -0.9046657085418701, 1.3351256847381592, 0.10722585767507553, 0.024423299357295036, -2.5642991065979004, 2.0513083934783936, -1.4966362714767456, 0.19377191364765167, 0.2211410403251648, -0.0990300104022026, -0.8386505842208862, 0.23122185468673706, 1.570652723312378, -1.0141849517822266, 0.8168031573295593, 2.0038421154022217, 1.3750561475753784, 0.023380780592560768, 0.17679482698440552, -0.5840082764625549, -0.47184303402900696, -0.51401686668396, 0.47087401151657104, 0.018403757363557816, 0.06723274290561676, -0.20306356251239777, -0.15822190046310425, 0.8775358200073242, 0.6124066710472107, 0.018794819712638855, 0.96818608045578, -0.09479542076587677, -0.1859516054391861, 0.6456639170646667, 0.02788991667330265, -1.7644983530044556, 0.16743923723697662, 1.6343250274658203, -0.25967463850975037, -0.1898432821035385, -0.2980026304721832, 0.6852172017097473, 0.33759891986846924, 2.8716776371002197, -0.22480806708335876, -0.7442190051078796, 0.395579069852829, -4.815954208374023, 0.11301465332508087, 0.021411044523119926, 0.08236230164766312, 0.09460195899009705, -0.1027514785528183, 0.28152650594711304, 0.5936724543571472, 0.2810978591442108, -0.05677567422389984, -0.3162476122379303, -0.26141998171806335, -0.9447765350341797, -0.19996336102485657, 2.6120359897613525, 0.08669501543045044, 0.018317079171538353, 2.1661698818206787, 0.054675158113241196, -0.0016929148696362972, -0.20817910134792328, -0.7274177074432373, -0.4693273603916168, 0.02214762568473816, -0.09491485357284546, -0.046958308666944504, 0.144881010055542, -0.03331518918275833, -0.9457836747169495, 1.8082585334777832, 0.594480574131012, 0.23787367343902588, 0.8695022463798523, 1.677621841430664, 0.31754177808761597, -0.8397479057312012, 0.3812090754508972, -1.2571592330932617, 0.0526261068880558, 0.11007292568683624, -0.9489515423774719, -0.06191316992044449, 1.5426586866378784, -0.34711557626724243, 0.06136851757764816, -0.22558066248893738, -0.46142399311065674, -0.1533227264881134, 3.6120572090148926, 0.5331622362136841, -0.10585083067417145, -0.9656715989112854, -0.8349859118461609, -0.3723508417606354, -0.029384562745690346, -0.29482194781303406, -0.44749343395233154, -3.408501625061035, -0.141971156001091, -0.7466832399368286, 0.23031899333000183, -0.26597338914871216, 1.056007742881775, -0.12731219828128815, 0.7487523555755615, -1.0546565055847168, -0.27996063232421875, 0.34668776392936707, 2.426663875579834, 0.029934652149677277, -1.4066556692123413, 0.024718550965189934, -0.703079342842102, -0.7506296634674072, -0.8581192493438721, -2.370302677154541, 0.22800223529338837, -0.04192981868982315, 2.7360758781433105, 0.033817119896411896, 0.16181223094463348, 0.23926283419132233, 0.2580772042274475, 0.2234080731868744, 0.6660608053207397, 0.40247365832328796, 0.03305186331272125, -0.29430314898490906, -1.3272329568862915, 0.6004620790481567, 0.3857080340385437, 0.7196481823921204, 1.6002386808395386, 0.23212827742099762, -1.4912962913513184, 0.021681660786271095, 1.7938652038574219, 0.5140511393547058, 0.22932299971580505, -0.9023024439811707, 0.08356676250696182, -0.03704458475112915, 0.0810493752360344, 0.7769655585289001, 3.5637776851654053, 2.093674659729004, -1.9978035688400269, 0.2507053315639496, 3.352147340774536, -1.3324633836746216, 0.21128210425376892, -0.3182297348976135, -0.5222945213317871, -0.07870791107416153, -0.0024313859175890684, -0.038735706359148026, 0.0008827071287669241, -0.08893153816461563, 0.5076307654380798, 0.4204157590866089, -2.292740821838379, 0.2771928906440735, -0.14161820709705353, -0.7578499913215637, 0.12614226341247559, 0.06752313673496246, 0.5476440191268921, -0.12815453112125397, 1.465545415878296, -0.23113490641117096, 0.06878985464572906, 1.0884772539138794, 0.0005427580326795578, 0.048416655510663986, 0.00010199426469625905, -0.20177583396434784, 0.5822567343711853, 0.019884631037712097, 0.4875810444355011, -0.21985846757888794, -0.0997251346707344, -1.47960364818573, -0.26764756441116333, -1.8316218852996826, -0.10899728536605835, 2.5022356510162354, -0.002337418729439378, 0.12558326125144958, 0.1326877474784851, 0.0011443353723734617, -0.023763831704854965, -0.14056164026260376, 0.014829563908278942, 0.07775086909532547, 0.1754891276359558, -1.0651029348373413, -0.17247042059898376, -0.05098900943994522, 0.7964414358139038, 0.22670744359493256, -1.8720446825027466, 2.099095344543457, -0.5904785990715027, -0.8018481731414795, -0.4172179400920868, -0.22701382637023926, 1.5388981103897095, 0.5600435733795166, 0.14979636669158936, -0.3484499454498291, 0.042793646454811096, -0.8134728074073792, -0.16840475797653198, -2.3786966800689697, -0.34679457545280457, -0.7030391097068787, 0.3849775791168213, -1.3577313423156738, 0.02011467143893242, 2.612517833709717, 0.32720112800598145, 0.2041788548231125, -0.4008972644805908, -0.6468002796173096, -2.88130784034729, -0.14494642615318298, -1.236023187637329, 0.05757439136505127, 0.1759852170944214, -0.6867038011550903, 0.1859862208366394, -1.7264643907546997, -1.2689918279647827, -0.10113935172557831, -0.004643154330551624, -0.010655632242560387, 0.5711852312088013, -0.9069182872772217, 0.3291785418987274, -0.030936378985643387, -1.341666340827942, 0.004530979320406914, 0.19502952694892883, 0.45787322521209717, -0.3259996473789215, -0.2518797814846039, -0.43178048729896545, 1.674979329109192, 0.1328158974647522, 1.2604540586471558, -0.9189143180847168, 0.08837392181158066, -0.5919453501701355, -0.11833516508340836, -0.3233366310596466, 0.6061831116676331, 0.2563760280609131, -2.0667507648468018, 0.31748074293136597, 0.44095420837402344, 1.0161588191986084, 0.2909596562385559, -0.2252107709646225, -0.062087271362543106, -0.8631947636604309, 0.045352302491664886, -0.8003485202789307, -0.41702428460121155, 1.1770198345184326, -0.26061010360717773, -0.40434977412223816, -5.2724480628967285, -2.3894007205963135, 0.25340497493743896, 0.661663830280304, -0.1987699568271637, -0.3883930742740631, -0.33033230900764465, -0.003947860095649958, -0.5832035541534424, -0.07463151961565018, 0.4081992506980896, 0.42684364318847656, -0.113865926861763, -1.5299887657165527, 1.5469260215759277, -0.014194786548614502, 0.8329606056213379, 0.41252070665359497, -0.04997296258807182, -1.4317266941070557, 1.0377135276794434, -0.32989025115966797, -1.0011425018310547, 1.2189216613769531, 0.052943143993616104, 0.15043891966342926, -2.72017502784729, 0.07401175051927567, -0.06236906349658966, 0.7698059678077698, 0.040782492607831955, 0.751535952091217, -1.3727493286132812, -0.7377707362174988, -0.6008864045143127, 0.15575195848941803, 0.04090811312198639, 0.01903083175420761, 1.398851990699768, 0.29887062311172485, 0.13515017926692963, -0.28289923071861267, -0.07640340924263, -1.8080663681030273, -0.5328502655029297, 0.48176345229148865, 2.443371534347534, 0.5587632060050964, 0.08157279342412949, 0.14120474457740784, -0.4703430235385895, 0.5273562073707581, -1.4133098125457764, 0.08986229449510574, 0.7215001583099365, 0.7094098329544067, -0.0799437165260315, -0.1644422560930252, 0.01580222323536873, 0.4805998206138611, -0.13903672993183136, -0.23322869837284088, 0.4634621739387512, 0.14426814019680023, -0.4740324914455414, -0.08144097030162811, 0.32417529821395874, 0.09790287166833878, -0.15065884590148926, -0.07449780404567719, -0.26440995931625366, -0.026061885058879852, -0.23798641562461853, 0.18353122472763062, 0.6755527853965759, -0.39163637161254883, 1.2129325866699219, -0.5352902412414551, 4.531235218048096, -0.4690679609775543, 0.14137548208236694, 1.4386018514633179, 0.07196356356143951, -0.7512866854667664, -0.03413833677768707, -5.562379837036133, -0.2505303621292114, -0.34786224365234375, 0.16338714957237244, -0.1295575499534607, 0.13792365789413452, 0.6461146473884583, 0.06977580487728119, 0.04632872715592384, -0.5706726312637329, -1.3501180410385132, -1.6633994579315186, -2.4222424030303955, 0.6058883666992188, -0.2133067101240158, 0.04479905962944031, 1.3535157442092896, -0.20105083286762238, -0.050372641533613205, -0.07327272742986679, 0.04576510190963745, 0.7881612777709961, 1.3613301515579224, 0.06776408851146698, 1.0148894786834717, -0.08411642909049988, -0.26572084426879883, 0.03335312753915787, -0.1226699948310852, -0.049792878329753876, 0.022569354623556137, -0.8571723699569702, -0.15072880685329437, -2.1280999183654785, 0.015782833099365234, 0.015127332881093025, -0.5349466800689697, -1.4297691583633423, 0.5955013632774353, -0.03452262282371521, 0.3875521123409271, 0.13315148651599884, -0.8757457733154297, 0.5548092126846313, -0.02977604791522026, 0.8329980969429016, -0.08748050034046173, -1.5986764430999756, 4.566346645355225, -1.1406044960021973, 0.9888591170310974, 0.17576225101947784, -0.004261905327439308, 3.1613149642944336, -0.15675745904445648, 1.2182903289794922, -0.06922892481088638, 0.6082821488380432, 1.760522723197937, -0.5914502739906311, -0.09987577050924301, -0.12558427453041077, -0.8812411427497864, -0.09369052946567535, -1.7993956804275513, -1.0666742324829102, -0.04514877125620842, 0.08639711886644363, 0.5030723214149475, 0.7852902412414551, 0.7201532125473022, 0.5531612038612366, 0.0061234538443386555, -0.028905168175697327, 0.7636716365814209, 1.0349229574203491, 0.03510812669992447, 3.1334426403045654, -0.0020689840894192457, 0.6362431049346924, 0.06595652550458908, 0.14574016630649567, 0.18071509897708893, 0.010065748356282711, 1.7429602146148682, 0.12331951409578323, -1.819510579109192, 0.05249907076358795, 0.690301239490509, 0.7944542765617371, 0.3943893611431122, -1.891357660293579, -1.1346280574798584, -0.06859365850687027, -0.864162266254425, -7.17793227522634e-05, -0.2905305325984955, -0.002642544684931636, -0.7669370770454407, -1.3217275142669678, 1.2454454898834229, 0.5778658390045166, 0.2835327088832855, -3.717499017715454, 0.8778007626533508, 2.338003396987915, -1.750598430633545, 0.5200923085212708, -0.01554965227842331, 2.4412925243377686, 0.690654993057251, -0.3502279222011566, 0.0870465487241745, 0.009009411558508873, 4.486656188964844, -0.9704424738883972, -0.876763641834259, 0.16058260202407837, 0.2808322310447693, -0.0017982613062486053, -0.7652460932731628, 0.21083174645900726, 0.1826065331697464, 0.03184884414076805, 0.7051241397857666, -1.1198034286499023, 0.3553360104560852, 0.8899247050285339, -1.0569907426834106, -0.5612272620201111, 0.039693910628557205, -0.025885354727506638, 0.8980619311332703, -0.14929962158203125, 0.15500949323177338, -0.005459954030811787, 0.6184663772583008, 0.08695244044065475, 0.28851109743118286, 0.3448590338230133, 0.23175674676895142, 1.21885347366333, -0.1792498528957367, 0.08166123181581497, 0.5354795455932617, -5.074057102203369, 0.15084242820739746, -0.038260363042354584, -0.35505568981170654, 0.049009136855602264, 0.4059852361679077, -0.029349403455853462, 0.02250487543642521, -0.09664208441972733, 0.025249416008591652, -0.1326013058423996, -2.202859878540039, 0.0956960991024971, -0.1389576643705368, -0.43123379349708557, -0.32802337408065796, -0.3847081661224365, 0.37496525049209595, 0.24893756210803986, 0.5548891425132751, -0.8720042705535889, 0.4149733781814575, -0.009974193759262562, 1.2576366662979126, 0.20277391374111176, -0.04926202818751335, -0.005124174524098635, 1.5424243211746216, 0.4326056241989136, 1.03099524974823, -0.0024188391398638487, -0.23757772147655487, 0.7672681212425232, 0.836655855178833, -0.11506444215774536, -0.7042976021766663, 0.2110106647014618, -0.056847091764211655, -0.046962957829236984, 0.03698509559035301, -0.5145936012268066, 0.262474000453949, 0.3073371946811676, -0.4422937333583832, -1.2435779571533203, -1.3181670904159546, -1.6728570461273193, -0.045366764068603516, 0.5730414986610413, -0.015065164305269718, 0.6794675588607788, -0.5567512512207031, -0.3671492040157318, 0.19521352648735046, 0.053011201322078705, -0.045617688447237015, -3.905968189239502, -0.27224043011665344, -0.12488458305597305, -2.6902647018432617, -1.631800889968872, 0.050911303609609604, -0.15875278413295746, 0.1370226889848709, 0.8505343794822693, -1.3646200895309448, -0.08690591156482697, 1.1245839595794678, 0.2719475030899048, -0.021396681666374207, -0.35836541652679443, -1.8827203512191772, -0.849539577960968, -0.6372487545013428, 0.4301310181617737, -0.18409283459186554, 1.0319602489471436, -0.09348082542419434, 0.26209756731987, -1.0254383087158203, 0.4769492447376251, -0.5229211449623108, -0.48081323504447937, 0.4190860688686371, -0.46582257747650146, 3.929921865463257, -1.3785812854766846, 0.5219761729240417, -0.8136224150657654, 2.0888495445251465, 0.8487129211425781, 0.005469504743814468, -0.18095645308494568, -0.37132614850997925, 0.22660529613494873, 0.07812663912773132, 0.05086345970630646, -0.006730874069035053, -0.11178727447986603, 1.1716225147247314, -2.3457770347595215, 0.5182946920394897, -0.44223636388778687, 0.014179538004100323, -0.5899890661239624, 1.280914545059204, -0.007896767929196358, 0.03492208197712898, 0.6488931775093079, 0.09445765614509583, -0.172712504863739, 0.4040745496749878, -1.2981873750686646, -0.4734640121459961, -0.7678885459899902, -0.2743293046951294, -0.41436952352523804, -0.9823846817016602, 0.0010798238217830658, 1.439660906791687, -0.2445223480463028, -0.18707497417926788, -0.9040084481239319, -0.7105383276939392, 0.0585796944797039, 1.0420126914978027, 0.46775922179222107, -0.6879050135612488, 0.25327545404434204, 0.1375008374452591, -0.11936318129301071, -0.159340962767601, -1.5245060920715332, -0.8198133111000061, -0.06465267390012741, 0.13674534857273102, 0.9167357087135315, 0.4883478283882141, -0.2419750839471817, 0.23695126175880432, 0.03578370064496994, 0.05961978808045387, -2.618847131729126, 0.014507189393043518, 0.023456985130906105, -0.3148044943809509, 3.692765712738037, -0.2305355668067932, 5.6018242835998535, 0.003620538394898176, 0.025611840188503265, -1.8184489011764526, 0.19234155118465424, 0.4612168073654175, -0.5093320608139038, -0.09290096908807755, 0.1988479644060135, 0.27126777172088623, -0.2604278326034546, -1.4043258428573608, 0.04317157343029976, 0.40091797709465027, -0.6300699710845947, -0.3247588574886322, -0.11262532323598862, -0.43736380338668823, -2.3188726902008057, -0.13099326193332672, -1.3595978021621704, -3.6384127140045166, -0.0019345983164384961, 0.09375835955142975, 0.013006195425987244, -2.3899950981140137, -0.553980827331543, -1.3559606075286865, -0.5831069946289062, -0.011837033554911613, -0.3248138129711151, -0.28520041704177856, 1.5472058057785034, -0.5824429392814636, 0.593713641166687, -0.7826665043830872, -0.8414552211761475, -0.23484931886196136, 0.8439655303955078, 1.0194091796875, 2.8867907524108887, -0.9492087960243225, 0.34613513946533203, 0.11998310685157776, -2.565920114517212, 0.3373936414718628, 0.13915342092514038, -0.23177982866764069, -0.05345439165830612, 0.2495671659708023, 0.7613396048545837, 0.00625591492280364, -0.5672903060913086, -0.007852268405258656, 0.5083382725715637, 0.0624723881483078, 2.040961742401123, -0.6161051392555237, 1.156030297279358, 0.812121570110321, -0.0018772392068058252, 0.49646154046058655, 0.34488004446029663, -0.5778983235359192, -0.0588604174554348, -0.3807408809661865, 1.0064657926559448, -0.10622041672468185, -0.016344809904694557, -0.5195578336715698, 0.11586948484182358, 0.7772355079650879, -0.025306526571512222, -0.4117216467857361, 0.9057838320732117, 2.8821897506713867, -0.7502554059028625, -0.23728616535663605, -0.022070404142141342, -0.05209772288799286, 0.013195385225117207, -0.4326065182685852, 0.016321444883942604, 0.07545354962348938, 0.5755221843719482, -0.8949754238128662, 1.1090421676635742, -1.0945912599563599, -0.06666955351829529, 1.3021188974380493, -1.5932823419570923, -0.5690551996231079, 0.0379258394241333, -0.5864883661270142, 0.3141273856163025, 0.1428534984588623, -0.0284500140696764, -0.22148364782333374, -0.17585979402065277, -0.7620770931243896, 1.5901436805725098, 0.19651882350444794, 0.671088457107544, 0.034427542239427567, -0.12501682341098785, 0.6588745713233948, -0.020662976428866386, -0.005013961810618639, -0.20514973998069763, -3.0164740085601807, 0.1651066690683365, 0.36533084511756897, 1.2622674703598022, -0.07071582973003387, 0.8545536398887634, -0.1837771087884903, 0.2199653536081314, 3.138286828994751, -0.04129913076758385, 0.10298707336187363, -0.016618836671113968, 0.33339765667915344, -1.3956464529037476, -0.47182926535606384, -0.28539976477622986, 0.5106642246246338, 1.1467740535736084, 0.4589141011238098, 0.42424601316452026, -0.20276005566120148, 0.011295901611447334, -0.016092676669359207, 0.17745791375637054, 0.3353109061717987, 0.6602006554603577, -0.22986522316932678, -0.5577690601348877, 0.6135345101356506, 0.6181685328483582, 0.3891049325466156, 1.696935772895813, 2.3624801635742188, 0.1960073560476303, -2.0779216289520264, 0.2748081386089325, 0.49149343371391296, -0.00852824468165636, 1.092745065689087, -0.5933188796043396, 1.3131775856018066, -0.7651684880256653, -0.8844636678695679, -0.4428252577781677, 0.9781903624534607, 0.06653229892253876, -0.05100008845329285, -0.16536907851696014, -1.3412061929702759, -0.06800144165754318, -0.7841359972953796, 0.3270399570465088, -0.2545650005340576, 2.106821060180664, 0.4692153334617615, 1.4717903137207031, -0.346157968044281, -0.06789503991603851, 0.3600640594959259, 0.4833822548389435, -0.09718827158212662, -0.885059654712677, 1.9978642463684082, 1.0285953283309937, 0.030946338549256325, -0.19493116438388824, -0.7209170460700989, -0.012828928418457508, 0.02089751698076725, -0.036069370806217194, 0.3406240940093994, 2.810532569885254, 0.7158134579658508, 0.0643225833773613, 0.11969122290611267, -1.55327570438385, 0.6393589377403259, -1.231050729751587, -0.22647756338119507, -1.1841580867767334, 0.6410081386566162, 0.24925458431243896, 0.46196556091308594, -0.06114973500370979, -0.3677658438682556, -0.38372230529785156, 0.0607529915869236, 0.648010790348053, 0.08718995749950409, 1.065995216369629, 0.048635225743055344, -0.9902511835098267, 3.306273937225342, -0.03892577812075615, 0.02492721937596798, 0.1239103451371193, -0.4242187440395355, 0.1301468163728714, -0.044637393206357956, 4.983425140380859, 0.18655657768249512, 0.6330227851867676, 0.30387914180755615, 0.7437586188316345, -0.06749697029590607, -0.12720154225826263, -0.17821717262268066, 0.5388937592506409, -2.02548885345459, -0.037268608808517456, -0.004490565974265337, -0.3824315667152405, 1.2809044122695923, 1.537916898727417, 1.5855592489242554, -1.9680548906326294, -0.2630157470703125, -0.3722788393497467, 0.81705242395401, 0.0010105224791914225, 6.109157085418701, -0.01695222407579422, 0.532701313495636, -0.037670962512493134, -0.6124255061149597, -0.49678876996040344, 0.020885074511170387, 0.015395483933389187, 0.01394115574657917, 1.3189526796340942, -0.20271874964237213, -1.2937337160110474, 0.5311571359634399, -0.0635920912027359, 1.4904382228851318, -0.0059026381932199, 0.2834644913673401, -1.8069020509719849, -0.8948517441749573, -2.72324538230896, 0.20474085211753845, -1.0256305932998657, -0.08406878262758255, -0.41457605361938477, -0.9307914972305298, 0.6394535303115845, -0.01293167844414711, 0.4378662705421448, 0.16112825274467468, 1.3599989414215088, 0.21861998736858368, 1.0903202295303345, -1.6519227027893066, -1.9684113264083862, -1.6313741207122803, 1.1303354501724243, -0.21899987757205963, -3.7600910663604736, 1.0738307237625122, -0.09690244495868683, 0.09992056339979172, 0.06775878369808197, 0.32252463698387146, 1.0335266590118408, 0.27145814895629883, -1.4721757173538208, -0.590918779373169, 0.7981104850769043, -1.299981713294983, 0.03740285709500313, -0.046859610825777054, -0.005438541993498802, 1.5298572778701782, 0.6459239721298218, -0.1682509481906891, -1.4606293439865112, 0.017087042331695557, -0.4793894290924072, 0.0779089406132698, -1.5463662147521973, 0.39816147089004517, -1.4827041625976562, 1.7767561674118042, -0.34697234630584717, -0.5802043080329895, -0.07350201904773712, 2.4078850746154785, 0.593988835811615, -0.24293625354766846, 1.1900990009307861, -1.7199355363845825, 0.037499476224184036, -1.222675085067749, -1.6059238910675049, 0.026087332516908646, -0.21757091581821442, -0.43399250507354736, -2.527449131011963, 0.44811496138572693, 0.16937962174415588, -0.06815334409475327, 0.6455259323120117, 0.20845380425453186, 0.6420079469680786, -0.27044546604156494, 0.22886671125888824, 0.728073000907898, -0.9419786334037781, -0.9886812567710876, -0.20057269930839539, 0.0015546433860436082, 0.3321606516838074, -0.3434695303440094, -0.002401992678642273, 0.9453471302986145, 0.07083398848772049, -0.01759904809296131, 1.6884040832519531, 0.10648057609796524, 1.4241427183151245, -2.5302884578704834, 2.6777846813201904, -0.20792730152606964, -1.9724699258804321, 1.6272660493850708, 0.5378081202507019, -0.02578711323440075, 0.1619410663843155, 2.2293148040771484, 0.4085863530635834, -5.219864368438721, -0.03766162320971489, 0.09790057688951492, 0.030844196677207947, 0.4420730769634247, -0.09609761089086533, -0.042070068418979645, 1.269761562347412, 0.12627336382865906, -0.0014833681052550673, 0.050632599741220474, -0.2116493284702301, -0.9926763772964478, -0.4000312387943268, 2.5642499923706055, -0.24955148994922638, -0.15721191465854645, 0.012492970563471317, 0.22321204841136932, 0.26930510997772217, 1.1764837503433228, -0.24520158767700195, 0.18239982426166534, 0.04781470447778702, -0.10565289109945297, -0.15834951400756836, 0.5799014568328857, 0.49646270275115967, 0.058857399970293045, -0.34707963466644287, -0.30335623025894165, -0.3734954297542572, 0.2540346086025238, 0.28963154554367065, 0.41080600023269653, 0.559641420841217, -0.2949463725090027, 1.3799742460250854, -1.4703904390335083, 2.9848759174346924, -0.3631778061389923, -0.5581014752388, 0.35469841957092285, 0.34830242395401, 2.4738411903381348, 1.2275519371032715, -0.42885899543762207, 0.2506698668003082, 3.813459873199463, -0.1475401520729065, -0.410364031791687, -1.0407183170318604, -0.9041491746902466, -0.5787653923034668, -0.36650192737579346, -0.06832229346036911, -2.531421184539795, -0.364819198846817, -0.06914570182561874, 1.719362497329712, -0.7835715413093567, -0.04079143702983856, 0.017769787460565567, -1.142137885093689, -0.07182525843381882, -1.142017126083374, 0.666844367980957, 0.21642200648784637, -0.1210671067237854, -2.7760841846466064, -0.17543485760688782, -0.36373409628868103, 0.17062485218048096, 0.001459023100323975, -0.29145491123199463, 1.0252619981765747, 0.2833862006664276, 0.35306882858276367, -0.6786168813705444, -0.6406434178352356, 0.1952270269393921, 0.11549105495214462, -0.007349452469497919, 0.060072652995586395, -2.4806418418884277, 0.00607577757909894, -3.568596124649048, 0.06273046880960464, -0.07030652463436127, -0.03977503255009651, -1.2054413557052612, -1.151344895362854, 1.5072598457336426, 3.8230628967285156, 0.04461580887436867, -0.4467894434928894, 0.01986139453947544, 0.11425404995679855, 0.20788241922855377, -0.4089520275592804, -0.1456909328699112, -0.1694662868976593, 0.15528880059719086, 0.7904000282287598, -0.21885287761688232, 0.44339272379875183, -1.1874531507492065, 0.28579115867614746, -0.028527146205306053, 1.9594764709472656, -0.12436726689338684, -0.9121894240379333, 0.013713255524635315, 0.35805806517601013, 0.6648412942886353, -2.2410027980804443, -0.46832841634750366, -0.3675605058670044, -1.3729716539382935, 0.1479036509990692, -0.8743868470191956, -0.026168093085289, -0.006396065000444651, -0.03455425426363945, 0.9475147724151611, -0.20731133222579956, 1.304923415184021, -0.02068699523806572, 0.7289494872093201, -0.019835716113448143, -0.031024513766169548, -1.452170491218567, -1.5289227962493896, -1.114540457725525, 0.25033751130104065, -2.686047315597534, 0.18063460290431976, -1.0802606344223022, 0.00015068928769323975, 1.061057209968567, 0.5927456021308899, 0.3530762493610382, 1.6421518325805664, 0.29003340005874634, -0.5693646669387817, -1.0750290155410767, 0.4132686257362366, 0.5710107684135437, -0.14197076857089996, -0.0787522941827774, 1.4738361835479736, 0.06218352168798447, -0.10169415920972824, 0.38723957538604736, -0.00605129124596715, -0.0012333402410149574, 0.316050261259079, -1.0576101541519165, -0.3846862018108368, -0.040045104920864105, -0.14902713894844055, -1.5220872163772583, -0.16622431576251984, -0.023856766521930695, 0.03121728077530861, 0.5354907512664795, 0.7664180397987366, 0.1774265617132187, 0.17035943269729614, 0.6556396484375, 0.5955058336257935, -0.01010535191744566, 0.4211493134498596, 2.65533447265625, 0.05477062612771988, 0.3594261705875397, 0.0640602558851242, -0.2825350761413574, 0.14648625254631042, 1.7809405326843262, 0.879655122756958, -0.021738026291131973, -0.025104770436882973, 0.8638612031936646, -0.061541177332401276, -1.450805902481079, -0.3974727690219879, 1.0537121295928955, 1.6340886354446411, 0.045482903718948364, -1.3161424398422241, 0.4171193838119507, -0.2265772819519043, 0.09229409694671631, 0.7367320656776428, -0.1493498533964157, 0.09446984529495239, -2.558763027191162, -1.3377403020858765, 0.020815350115299225, -0.027274513617157936, -2.0463802814483643, 1.2809618711471558, 0.09036588668823242, -0.1903454065322876, -0.23737530410289764, 0.2128472477197647, -0.06101185455918312, -0.48857229948043823, -0.020220037549734116, 0.8200734257698059, -0.25125783681869507, 0.09024904668331146, 0.45961058139801025, -0.27779585123062134, -0.003730745054781437, -6.034971714019775, 0.09136606007814407, -0.2823888957500458, 0.2569885849952698, 0.7364028096199036, 0.3745627701282501, 0.11038623750209808, -0.06537636369466782, -1.7774693965911865, 0.2537069022655487, 0.0045947786420583725, 0.27906277775764465, 1.6162819862365723, 0.45668143033981323, 0.7204481959342957, 1.5797001123428345, -0.7885978817939758, 0.010168765671551228, 0.09960780292749405, 0.14499656856060028, 0.34919825196266174, 0.009987951256334782, -0.015221550129354, 0.14150671660900116, -0.8887475728988647, 0.06434985250234604, 0.09314502030611038, 1.2488582134246826, -1.8648585081100464, 0.03588132560253143, -0.12680315971374512, -0.0170542411506176, -1.8510019779205322, 0.10271327197551727, 0.8476611375808716, 0.43747955560684204, -0.036035582423210144, 0.043280553072690964, -0.010497004725039005, -0.10332262516021729, -0.050200823694467545, 0.7749465107917786, -0.811708390712738, -1.576521873474121, -0.7052106857299805, 0.7994214296340942, -0.19468334317207336, 0.023350205272436142, 0.33779624104499817, 0.5254555344581604, -0.0972253605723381, 0.07412227988243103, 0.10132937133312225, 0.18167951703071594, -0.47303786873817444, 0.8750389814376831, -0.7782092690467834, 0.11270182579755783, 0.13053490221500397, 0.6376016736030579, -2.2839744091033936, 0.9563882350921631, -0.6355693340301514, 0.004516046494245529, 0.32690948247909546, -1.0565311908721924, -0.4284619092941284, 1.5405689477920532, -0.1761951893568039, 0.22897866368293762, -0.7771416902542114, 0.5825561285018921, 0.88008052110672, -0.6729037761688232, 1.2072243690490723, -3.380032539367676, 0.7855452299118042, 0.5767596364021301, -1.2507224082946777, -1.04148268699646, 0.831816554069519, -0.006818404886871576, 0.19986028969287872, 1.1429272890090942, -1.0001565217971802, 0.07463591545820236, -1.3568881750106812, 0.322884738445282, 0.061800990253686905, 0.4041048288345337, 0.12648120522499084, -0.37741559743881226, -0.8941963315010071, 0.07330480217933655, -1.6640485525131226, -0.04189637675881386, -0.26883578300476074, -0.23922871053218842, 0.19255216419696808, 2.1116714477539062, 0.8646916151046753, -0.3723140358924866, 0.005125206429511309, -0.4412354528903961, 0.6343607902526855, 0.002240621717646718, 0.05190784111618996, -0.45549842715263367, 2.298386812210083, 2.0182974338531494, 1.051888108253479, -0.15553876757621765, 0.03328520804643631, -0.017705390229821205, -0.6998550891876221, 1.4186848402023315, -1.0000348091125488, 0.611175537109375, -0.761771559715271, -0.8818244934082031, 0.10376142710447311, -0.020069804042577744, 0.48889684677124023, -0.28268665075302124, 0.1412699818611145, -0.58719801902771, -0.42485034465789795, -0.5690693259239197, -0.10763586312532425, -0.2417701631784439, -0.15777666866779327, 0.6839241981506348, 1.0112590789794922, -0.2062491923570633, -0.019949989393353462, 1.01609468460083, -0.05353231728076935, -0.45140013098716736, 1.1670629978179932, -0.6911106109619141, 0.8987969160079956, -0.6109169721603394, -1.345786452293396, 0.3047926127910614, -2.6888160705566406, -1.3229870796203613, -0.5358954071998596, 0.4913531243801117, 0.7799898982048035, 0.06508419662714005, 0.07196009904146194, -1.0946646928787231, -0.23678477108478546, -0.06832278519868851, -0.15775027871131897, 0.43571096658706665, -0.2125849723815918, 0.24992060661315918, -0.14134247601032257, 0.16573470830917358, 0.7403910160064697, 0.16459514200687408, -1.4203253984451294, -0.43646666407585144, 1.5473730564117432, 0.0668383240699768, 0.045257002115249634, 0.32035955786705017, 0.0035331177059561014, 3.0460422039031982, -0.4145817756652832, 0.0033019124530255795, 0.18471591174602509, 0.18467777967453003, 0.2825365662574768, -0.47481730580329895, 0.8493618965148926, -0.32546159625053406, -0.8195186853408813, 0.13531899452209473, 1.3629331588745117, 0.8017728924751282, 0.07303951680660248, 1.050613284111023, -0.5058886408805847, 0.31972944736480713, 0.2456856518983841, -0.010189172811806202, 0.41902822256088257, -0.05876912176609039, 0.6628124713897705, -1.562684416770935, 0.33173230290412903, -0.4238087832927704, 0.12398333102464676, -1.9040753841400146, -0.10463409870862961, -0.09425992518663406, -0.5041225552558899, -2.8833558559417725, -0.036118656396865845, 0.4598168134689331, -0.9371246695518494, 0.8771407008171082, 0.0036276669707149267, -0.008872461505234241, -1.0658096075057983, 0.5863257646560669, 0.001589686842635274, -0.10421258956193924, -0.3750585913658142, 0.42126065492630005, 1.1302928924560547, 2.0136537551879883, 1.5153889656066895, -0.09598294645547867, -0.23481588065624237, -0.4579527676105499, -0.6452271938323975, -0.0016628222074359655, -0.008108090609312057, 0.534574031829834, 0.7584447264671326, 0.3721467852592468, -1.380224585533142, 1.355164885520935, 0.7717074751853943, 0.18234264850616455, -0.1754133701324463, 0.26269325613975525, 2.6314806938171387, -0.1024065837264061, -1.1407674551010132, -2.4713704586029053, -0.2190738320350647, -0.6674940586090088, -0.0765579342842102, -0.07202780991792679, -1.0706270933151245, 0.08032000064849854, -0.672183096408844, -0.09393306821584702, -0.009967547841370106, -1.0299673080444336, -0.7359009981155396, 0.2920868694782257, 0.42325595021247864, -0.4865247905254364, 0.06908763945102692, -1.6673732995986938, -1.105833888053894, 0.008902540430426598, 0.2208433598279953, -0.23072437942028046, -0.6181273460388184, 0.006524849217385054, 0.034920334815979004, 0.3355574309825897, -0.04457100108265877, 0.025384442880749702, -0.46399861574172974, 0.5137211680412292, -0.05355911701917648, 0.31619563698768616, -0.0998513251543045, -0.5506685376167297, -0.12511473894119263, 0.01105165109038353, -0.7449577450752258, 0.417060911655426, -0.08615253120660782, 0.01332930475473404, 0.5371488928794861, 0.172227680683136, 0.5365046262741089, -1.213343858718872, 1.428139328956604, -0.21089626848697662, 0.050603561103343964, 2.661867380142212, 0.21757441759109497, -0.13981766998767853, 3.3462905883789062, 0.1842942237854004, 0.6120176315307617, 0.029833875596523285, -0.303180456161499, 0.2755108177661896, -0.12805233895778656, 0.3884364664554596, -0.015288962982594967, -0.03585568070411682, 0.7786291837692261, -0.028246337547898293, -0.25054988265037537, 2.0375521183013916, -0.8147546648979187, -0.12720388174057007, -1.0905314683914185, 1.5068148374557495, -0.17011013627052307, 1.3931529521942139, 2.032485246658325, 0.16386725008487701, 0.015585796907544136, 1.0851519107818604, 0.6647675037384033, -2.8746800422668457, -1.4801594018936157, -1.6001266241073608, 0.048701368272304535, 0.04431648552417755, 0.5806873440742493, -0.013867942616343498, -0.2919750511646271, -0.04386341571807861, -0.6854552626609802, -0.9119502902030945, 0.09009460359811783, 1.9151241779327393, 0.4222434163093567, -0.059459418058395386, 0.054679881781339645, -0.009625323116779327, -5.748137474060059, -0.3800029456615448, 1.064804196357727, 1.3939718008041382, 0.07439858466386795, 1.0324807167053223, 2.212770938873291, -0.05221034213900566, -0.02367575094103813, 0.648388683795929, 1.0282670259475708, 0.0448269322514534, 1.0284475088119507, 2.0519280433654785, -0.042431801557540894, -0.3964957296848297, 1.0588233470916748, 1.0792784690856934, 1.3830565214157104, 1.1646795272827148, 0.20059676468372345, -2.286095380783081, 0.7632339596748352, -0.25975650548934937, -0.7998671531677246, -0.017994806170463562, -0.43123048543930054, 0.5423281192779541, -0.386721134185791, 1.3751295804977417, 0.37734586000442505, -0.44416120648384094, -0.4148136079311371, -0.23750299215316772, -0.16078419983386993, -0.30676740407943726, -0.6813197731971741, 0.34870773553848267, -0.5057339668273926, -0.8484697937965393, 0.19429536163806915, -0.6258315443992615, -0.1265759915113449, 0.8519085049629211, -0.24763232469558716, 4.368097305297852, 0.08629781007766724, 0.07532748579978943, 0.3095381557941437, -1.143682837486267, -2.2792739868164062, 0.6450130939483643, -0.3716563880443573, 0.08809439837932587, 0.1603943556547165, -0.002026789588853717, 0.011408267542719841, -0.39650267362594604, 0.20209276676177979, -0.39334315061569214, 0.6008850336074829, -0.21224352717399597, 0.2697363793849945, 0.6777058839797974, -0.5691118836402893, -0.008264172822237015, 0.022504689171910286, -0.9663246870040894, 0.09256116300821304, 1.4700093269348145, -0.3733392059803009, -0.7351573705673218, 0.03196975961327553, 0.025818560272455215, -1.0437685251235962, 0.3754144012928009, 0.36310771107673645, 0.30852973461151123, -2.356489419937134, 0.2568879723548889, 0.6721571683883667, 0.008559099398553371, -0.6556814908981323, -0.3109849989414215, -0.8344935774803162, 0.00028574588941410184, 0.7406132221221924, -0.7971388697624207, -0.38113778829574585, -0.4650844931602478, -0.7000635862350464, -1.1617028713226318, -0.3043379783630371, 0.4741617441177368, -0.9870232939720154, -0.7054572701454163, -1.4997681379318237, -1.0864218473434448, -0.127421572804451, -0.08525703847408295, 0.018669428303837776, -0.11132541298866272, -0.04258539155125618, -1.7642461061477661, 0.2951246500015259, -0.9689567685127258, 0.03752094507217407, 1.2770189046859741, -0.046885598450899124, 0.5470409989356995, 0.33579784631729126, 0.02699209563434124, -5.377204895019531, 0.6939811706542969, -0.41718897223472595, -0.37808355689048767, 0.0954492837190628, -0.23383581638336182, 0.23903129994869232, 0.69037926197052, -0.39827364683151245, 0.5294843316078186, 0.08993898332118988, -0.3183051645755768, -0.1139514297246933, -0.49091488122940063, 0.24304594099521637, 0.10121607780456543, 2.8611490726470947, -0.1917879581451416, 0.8067610859870911, 0.10046909004449844, 0.490270733833313, -0.03881034255027771, 0.6275295615196228, 0.06760770827531815, -0.17563669383525848, 1.0979094505310059, -0.6679356694221497, 1.6991827487945557, -2.544123888015747, -1.2919903993606567, 0.7054168581962585, -0.055914878845214844, 0.03533276170492172, -0.1939372420310974, -0.8911288380622864, -0.1346742808818817, -1.665894627571106, -0.03329944610595703, -0.033234741538763046, 1.7798733711242676, 0.02421809919178486, 4.017623424530029, 0.23938485980033875, -0.24527885019779205, 1.8916993141174316, -1.201040267944336, 0.4520200192928314, -1.216850757598877, 1.6313847303390503, -1.0895317792892456, 0.5887262225151062, -0.03200603649020195, -0.10555754601955414, -0.09146833419799805, 0.06066949665546417, -0.21213629841804504, -0.5899528861045837, 3.624962568283081, 1.3303335905075073, -0.7258365154266357, -1.070425033569336, 0.7390859127044678, -1.4910365343093872, -1.3422818183898926, 0.8085569739341736, 0.9543651938438416, -1.2636851072311401, 0.24751926958560944, -0.3096657693386078, 0.2720300257205963, 0.21428677439689636, 1.1639494895935059, 1.2678916454315186, -0.2773864269256592, 0.32552510499954224, 0.317380428314209, 0.44835811853408813, -0.2365785539150238, -0.33017027378082275, 0.24462449550628662, 0.02608826942741871, 0.12194404751062393, 0.5079148411750793, -0.6232286095619202, -0.11355241388082504, 0.00924269575625658, -0.007586590480059385, 0.46392759680747986, 0.4158044159412384, 0.20916078984737396, 0.015262436121702194, 0.05271027609705925, -0.05153966695070267, -0.13606195151805878, -0.02494502253830433, 1.2413502931594849, -0.4770464599132538, 0.013562846928834915, -0.5233570337295532, 0.3352636396884918, -1.6653902530670166, 0.5820888876914978, 0.044295016676187515, -2.001645803451538, -2.1919620037078857, -0.15504661202430725, -1.182737946510315, 1.4962881803512573, 0.5998477935791016, 0.07901239395141602, 0.19811105728149414, -0.16000451147556305, -1.7714149951934814, 0.5325897336006165, -0.4913230836391449, 0.526940107345581, -0.2897700071334839, -1.8879814147949219, -0.05683843791484833, 0.06154389679431915, 0.03985237702727318, 0.025747252628207207, 1.0981379747390747, -0.13156872987747192, 0.9064589738845825, -0.5598764419555664, 0.7050358653068542, 0.2933230400085449, 1.2110369205474854, -0.7850866913795471, -1.2094964981079102, -0.30361220240592957, -4.7210259437561035, -0.14961758255958557, 2.2234020233154297, -0.535560131072998, 0.11676213890314102, 0.11642405390739441, 0.15776261687278748, 0.2337113916873932, -0.39531779289245605, -0.33774587512016296, -6.463176727294922, -0.09755042940378189, -2.5993711948394775, -3.0179011821746826, 0.031160501763224602, -1.6234714984893799, 0.10097075253725052, -0.5551666021347046, 4.0180888175964355, 2.8315584659576416, -0.27996399998664856, -0.3681979775428772, 0.253135085105896, -0.11139999330043793, -0.07025300711393356, -0.10302820056676865, -0.00010272401414113119, 0.5075339078903198, 0.3414890468120575, 2.8152310848236084, -0.1854681670665741, -2.0241987705230713, -0.04079311713576317, 0.24799884855747223, -1.9690325260162354, -1.234581708908081, -1.3222743272781372, -0.43587589263916016, 1.2080433368682861, 1.6444923877716064, -0.1277078241109848, -1.4372479915618896, -0.6494836807250977, 1.5053128004074097, 1.0253899097442627, 0.31212136149406433, 0.4546493589878082, 0.11469323933124542, 0.0238194577395916, 0.008363429456949234, 0.011161452159285545, 0.005650785751640797, -1.929473876953125, -0.5266362428665161, -0.6953414082527161, 0.7099350094795227, 0.1169007197022438, 0.3992671072483063, 0.11183835566043854, -0.02783850021660328, 2.319162130355835, 0.010933377780020237, 0.15699197351932526, 1.2840640544891357, -0.4997478723526001, 1.2985084056854248, 0.09014248102903366, -0.6927463412284851, -0.19506238400936127, 2.2291955947875977, 0.4269121587276459, 0.0018235519528388977, -0.5244863629341125, -0.9333909749984741, -0.2169630080461502, 2.827300786972046, -0.2733466923236847, 0.35987332463264465, 0.5049542784690857, 0.4635534882545471, -0.005446321796625853, 0.0006490813684649765, -0.015265394002199173, 0.17430385947227478, -0.8409667611122131, 0.26379096508026123, -1.580147385597229, -1.2123782634735107, -0.020278463140130043, -0.4015969932079315, -0.3596425950527191, 0.49197569489479065, -0.44829922914505005, -0.063225656747818, -0.5722060799598694, -0.061145421117544174, -0.17095527052879333, 0.622117817401886, 2.737069606781006, 0.3553774356842041, -2.332339286804199, -0.3758068084716797, -0.13609899580478668, -0.06587126106023788, -0.01656370982527733, 0.9673002362251282, 0.04240617901086807, 0.34411531686782837, -0.9880694150924683, -1.0350062847137451, -0.11465201526880264, -0.7753110527992249, 1.2317070960998535, -0.15599282085895538, -0.012041937559843063, -1.8873560428619385, 0.3585061728954315, 1.4341338872909546, 0.45862290263175964, 0.05115662142634392, -1.7580392360687256, -0.09160169214010239, 0.1060962975025177, -1.4228798151016235, -0.44403353333473206, -1.4228965044021606, -0.5251480340957642, 1.6567118167877197, 0.818789541721344, 0.018507104367017746, 0.0334845669567585, -0.12246518582105637, -2.002027988433838, -0.006846448872238398, -0.1889621615409851, -0.7031111717224121, 0.731003999710083, 0.17527824640274048, -0.5632168054580688, 0.051733192056417465, 1.254381537437439, 0.09982117265462875, 0.08341021090745926, -1.8019013404846191, 0.7459738254547119, -0.1989610344171524, 0.2326282411813736, 0.27000850439071655, -0.1822165846824646, -1.8046437501907349, 2.238595962524414, -0.10007171332836151, 0.02805349789559841, 0.2717129588127136, 3.913522481918335, 1.9165014028549194, 0.06492216885089874, 0.09727717936038971, 0.09550060331821442, 0.45388901233673096, -0.5903934836387634, -0.021449239924550056, -2.3787102699279785, -0.21199482679367065, -1.1209771633148193, 1.0954903364181519, -0.32952529191970825, 0.27226176857948303, 0.08619695901870728, -0.19397655129432678, 0.925566554069519, 0.07721278071403503, -0.021526867523789406, 0.17787498235702515, 0.22778917849063873, -2.2223219871520996, -3.415088176727295, 0.12326875329017639, -0.18519681692123413, -0.13172170519828796, -0.47647178173065186, -0.08096186071634293, 0.1302398145198822, -3.975576639175415, 0.011390695348381996, -0.22765761613845825, 0.1118873581290245, -1.674141764640808, 1.000851035118103, 0.13470429182052612, 0.0880231037735939, -0.1864970624446869, 0.0019078569021075964, 0.5620213747024536, -0.09715422242879868, -0.4958086609840393]], "dtype": "float32"}, {"__nd__": [[-68.6563949584961, 6.657850742340088, -24.821826934814453, -35.80926513671875, 19.24286651611328, -20.404254913330078, -13.352472305297852, 13.76671028137207, -37.20726013183594, 2.850569486618042, 4.610074520111084, -63.74360656738281, -9.465441703796387, 6.5977373123168945, 16.048580169677734, 23.50065803527832, -36.053443908691406, 25.72817039489746, 20.397483825683594, -35.451820373535156, -43.939605712890625, -19.022422790527344, -31.2947998046875, -21.195085525512695, -50.26594161987305, 9.56098461151123, -21.82879638671875, -35.38774490356445, -5.671771049499512, 39.77132034301758, -33.33884811401367, -11.821941375732422, -28.69664764404297, 33.313262939453125, 8.428099632263184, 18.99287986755371, -16.46510124206543, 20.43372344970703, -31.760602951049805, 5.864928245544434, 6.972704887390137, 42.442840576171875, -66.48257446289062, 0.21738648414611816, 0.004344165325164795, 14.807193756103516, 19.85692596435547, 18.996849060058594, 55.859031677246094, 65.11898040771484, 4.413477897644043, 4.964840888977051, -13.434406280517578, -33.75157165527344, 28.491939544677734, -51.77384948730469, 3.4367480278015137, 48.121402740478516, -87.12480163574219, -51.74578094482422, -20.302366256713867, -22.215713500976562, -26.18920135498047, 26.272403717041016, 35.927833557128906, -20.703832626342773, 46.896827697753906, -27.785383224487305, -1.5514963865280151, -30.996339797973633, -12.13619613647461, 28.29398536682129, -63.327674865722656, 7.89424467086792, -23.27778434753418, 10.984466552734375, 0.01247107982635498, -32.3272705078125, 30.2596492767334, 10.076309204101562, -43.23082733154297, 31.571556091308594, 4.7094902992248535, -19.039859771728516, -17.92250633239746, 52.199462890625, 15.534797668457031, -6.726744651794434, -10.218591690063477, -13.013391494750977, 15.277274131774902, -8.76807689666748, 6.361155986785889, -2.9604098796844482, -17.755760192871094, -3.9476566314697266, -38.189395904541016, -15.672749519348145, -64.87474822998047, 17.14261245727539, 13.431976318359375, 52.802589416503906, 58.42443084716797, -10.090646743774414, -18.695274353027344, 9.40931224822998, 72.69044494628906, 76.64997100830078, -6.92508602142334, 14.161182403564453, 40.211891174316406, 14.522378921508789, 12.879632949829102, -36.64897537231445, -19.821609497070312, -21.13613510131836, -69.32305908203125, -27.78603172302246, 42.76814270019531, 15.169614791870117, 3.888838529586792, -6.843558311462402, 21.80582618713379, -31.481964111328125, -30.322006225585938, -1.8975270986557007, -6.343863010406494, 5.3670654296875, -22.441503524780273, -34.894630432128906, -8.749113082885742, 21.28681755065918, -22.626300811767578, -3.5482637882232666, 21.029144287109375, -55.322052001953125, -11.780794143676758, -16.067089080810547, -25.1754207611084, 6.416477203369141, 4.078633785247803, 15.024354934692383, -44.405487060546875, 10.427108764648438, -6.53695011138916, -13.831490516662598, 49.063663482666016, -9.022192001342773, -33.30642318725586, 13.363106727600098, -53.230224609375, 36.865150451660156, 39.46042251586914, 49.91910171508789, 14.27121353149414, 18.936237335205078, 12.87302017211914, -27.453622817993164, 20.872386932373047, -18.28243064880371, -33.54896545410156, -25.099624633789062, -0.25595664978027344, -12.194843292236328, -14.469645500183105, 37.4564323425293, 15.29770565032959, -19.005708694458008, 29.828088760375977, 51.01445007324219, -26.249807357788086, -8.989494323730469, 56.094268798828125, 9.881561279296875, -24.443368911743164, 35.112586975097656, 3.712759494781494, -12.341404914855957, 15.559703826904297, 16.312103271484375, -38.58517837524414, -4.639212608337402, -24.304203033447266, 25.76780891418457, -41.48183059692383, -1.5912107229232788, -45.614261627197266, 6.689496994018555, 36.12002182006836, 7.96201229095459, 0.3939921259880066, -2.2510108947753906, 44.10394287109375, -39.211204528808594, 29.728832244873047, -14.643823623657227, -8.179645538330078, -34.604881286621094, 9.729941368103027, 12.583303451538086, -15.093047142028809, -2.552371025085449, 1.9674155712127686, 54.33891677856445, 13.756208419799805, 73.94402313232422, 11.431246757507324, 62.531463623046875, 14.295825958251953, 0.9111850261688232, -0.9383188486099243, -62.7642822265625, 12.717020988464355, 8.402541160583496, 33.68756103515625, 15.703997611999512, 66.20797729492188, 11.26336669921875, 10.18046760559082, 18.971731185913086, -22.519872665405273, -5.597722053527832, 9.654061317443848, 4.295379161834717, -9.374438285827637, 14.338502883911133, 20.99164581298828, -51.97197723388672, 1.5002222061157227, 28.39447021484375, 5.452839374542236, 6.850628852844238, -50.176143646240234, 31.50664710998535, -23.576374053955078, 38.10163116455078, 29.33875846862793, 34.14161682128906, -31.11022186279297, 18.990610122680664, 25.173891067504883, -6.684628486633301, -25.023164749145508, -40.97782897949219, 27.423194885253906, 4.48882532119751, -20.613845825195312, -19.38437271118164, 8.677359580993652, -22.47571563720703, -9.987509727478027, -26.84697914123535, 3.9802963733673096, -0.622376561164856, 59.47161102294922, 29.529342651367188, -11.510955810546875, -6.917440414428711, -97.70014190673828, -40.53734588623047, 32.753990173339844, -12.189689636230469, 8.713768005371094, 16.19445037841797, -30.21310806274414, -21.666011810302734, 32.99818801879883, -10.063407897949219, 31.81270980834961, -21.126911163330078, -8.390985488891602, -5.280573844909668, 15.048160552978516, -9.728185653686523, 4.452545166015625, -0.5087141394615173, 3.146125555038452, 9.649356842041016, -7.533218860626221, -25.249418258666992, 31.555219650268555, -3.806013584136963, 28.368558883666992, 33.38056945800781, 27.846359252929688, -14.688909530639648, -13.66275691986084, -27.20647430419922, 21.893354415893555, 22.397499084472656, 46.53418731689453, -30.97687530517578, -76.00479888916016, 3.8930258750915527, 14.394723892211914, -24.803604125976562, -19.867839813232422, 0.3209081292152405, -40.605445861816406, 12.510957717895508, -17.06514549255371, 50.7143440246582, 25.692838668823242, 34.62046813964844, -5.770567893981934, 32.95234680175781, 5.353235244750977, 70.07986450195312, -8.00609302520752, -6.246910572052002, -5.879648208618164, -31.545324325561523, -1.9124984741210938, 20.60015296936035, 42.99178695678711, 48.76074981689453, -25.707796096801758, -16.491090774536133, -28.823017120361328, -10.74089241027832, 75.76569366455078, -9.269731521606445, -3.793881416320801, -45.86569595336914, -6.863338470458984, 20.115446090698242, -8.391107559204102, 10.615763664245605, 17.01530647277832, -9.695611000061035, -21.657730102539062, 5.582779407501221, 11.58145523071289, 11.337259292602539, -23.550954818725586, -14.988197326660156, 71.20732116699219, -49.58482360839844, 16.667848587036133, 9.157072067260742, 26.61992073059082, -42.940731048583984, -21.168840408325195, 29.143762588500977, -42.67085647583008, -41.268680572509766, -23.0875301361084, 1.4225239753723145, 42.848880767822266, -4.916590213775635, 7.5098958015441895, 30.35442352294922, 44.348812103271484, -26.9302921295166, -19.335704803466797, -26.8563175201416, -16.13095474243164, -19.92098617553711, -63.962562561035156, 36.620811462402344, -3.7541615962982178, 13.091374397277832, -35.161720275878906, -10.812603950500488, 17.487899780273438, -4.643446445465088, -14.340424537658691, 25.92726707458496, -1.5881906747817993, -28.33036994934082, -36.41485595703125, -2.8395638465881348, -11.937068939208984, 13.092788696289062, -20.955013275146484, 20.552932739257812, -2.5311217308044434, -51.867393493652344, -12.21041488647461, -16.215219497680664, -10.487730026245117, -8.085338592529297, -15.202607154846191, 61.63603210449219, -12.52039909362793, -47.4470100402832, -4.459681034088135, -30.72127342224121, 28.187376022338867, 11.899678230285645, 34.18938446044922, -73.07139587402344, -15.115396499633789, -18.299819946289062, -14.323073387145996, -18.847431182861328, -11.260668754577637, -29.710603713989258, 6.733925819396973, -28.536766052246094, 47.2337646484375, -25.69980239868164, 66.6685791015625, -15.272638320922852, -3.1646883487701416, -61.7603759765625, -39.623756408691406, -7.267704486846924, 35.97813415527344, -14.162057876586914, -4.98256778717041, 17.563215255737305, 20.752653121948242, -46.79944610595703, 45.22237014770508, 19.560298919677734, 3.344437837600708, 9.707173347473145, 43.07711410522461, 21.776845932006836, -4.1287126541137695, -43.06446075439453, -11.675046920776367, 3.240090847015381, -11.039724349975586, 55.923465728759766, 34.886192321777344, -7.797001361846924, 12.558554649353027, -18.852001190185547, 65.97962188720703, 8.199918746948242, 26.533565521240234, 4.411921501159668, 8.180437088012695, 9.154054641723633, -13.025185585021973, 5.820545196533203, -59.10711669921875, 61.354286193847656, 34.088836669921875, -14.159584045410156, 23.57067108154297, 10.416894912719727, 7.300715446472168, 13.608386039733887, -59.340946197509766, 17.703155517578125, 37.370792388916016, 16.649736404418945, -44.2819938659668, 20.11272430419922, 10.806739807128906, 35.91720962524414, 13.374337196350098, 45.735904693603516, 16.018360137939453, 30.92774200439453, 14.387612342834473, -4.479078769683838, -64.26121520996094, 16.026805877685547, 23.079368591308594, 17.120433807373047, 28.053749084472656, 17.966760635375977, 16.83995246887207, 15.987353324890137, -10.50574016571045, 33.44048309326172, -1.1787750720977783, -20.6544132232666, 43.738792419433594, 28.398176193237305, 17.300861358642578, 12.461029052734375, -39.10477066040039, -8.327982902526855, -22.87554359436035, 18.512060165405273, 5.700410842895508, 24.05750274658203, 7.770575523376465, -97.76832580566406, 42.322635650634766, -5.8500518798828125, -21.703598022460938, 38.00417709350586, -15.095916748046875, 37.22663116455078, 17.37893295288086, -21.188255310058594, 0.48669183254241943, 21.754955291748047, 0.17482642829418182, -20.355815887451172, 21.758712768554688, 40.76069259643555, -39.493316650390625, 30.984891891479492, -63.865699768066406, 43.48712158203125, -25.19313621520996, 8.872973442077637, -5.216240882873535, -24.327985763549805, -15.42444133758545, 7.40075159072876, -24.999622344970703, -19.331695556640625, -5.309209823608398, 10.670380592346191, -3.0263898372650146, 10.396398544311523, -43.561424255371094, 10.737451553344727, 12.1763277053833, 66.51570129394531, 18.747474670410156, -17.605064392089844, 13.908761978149414, -14.134916305541992, 54.99629211425781, -43.464176177978516, 57.85652160644531, 27.08209228515625, -3.0972647666931152, 52.42811584472656, -14.629942893981934, -12.095407485961914, 6.41884708404541, 15.136323928833008, -44.016685485839844, 10.549636840820312, 11.156213760375977, 25.22504425048828, -9.575972557067871, -32.93915939331055, -16.038326263427734, 66.04374694824219, -14.938624382019043, 38.71892166137695, -26.534753799438477, -6.952748775482178, -62.345333099365234, -3.6184957027435303, -49.76518630981445, 19.955575942993164, -41.363494873046875, -9.006617546081543, 5.576937198638916, -39.70110321044922, 45.195091247558594, -21.930397033691406, -22.748512268066406, -35.47191619873047, -45.59980010986328, 18.55592918395996, 2.326467990875244, 3.498276710510254, -54.438941955566406, -16.141422271728516, -9.184955596923828, -6.968879699707031, -16.325069427490234, -45.68040084838867, 0.9992177486419678, 11.675708770751953, 64.23283386230469, 8.374849319458008, -60.489044189453125, 20.894254684448242, -26.334779739379883, -71.4021987915039, -38.8421630859375, 15.779397010803223, -75.26812744140625, -21.42600440979004, -27.960472106933594, 4.7899956703186035, -36.124610900878906, -11.555726051330566, 36.017173767089844, 2.6157352924346924, -10.102049827575684, -50.37417984008789, 58.08578109741211, -3.464077949523926, -45.85283279418945, 42.285316467285156, 19.30004119873047, 37.08034896850586, 24.26926612854004, -52.16675567626953, -12.654082298278809, -12.668584823608398, -61.83403396606445, -12.572909355163574, 49.3121452331543, -1.4538785219192505, 12.00368595123291, 41.20817947387695, 23.209774017333984, 21.50522804260254, 0.47923335433006287, -25.679691314697266, 0.08823251724243164, 19.091459274291992, -19.474075317382812, 14.784381866455078, -23.581138610839844, 2.545419931411743, -54.9683723449707, 8.816612243652344, 23.08330535888672, 2.0143628120422363, -8.979806900024414, 33.29022216796875, 9.765731811523438, 7.9400763511657715, -22.082439422607422, -15.800863265991211, 31.587825775146484, 66.86820220947266, 29.118547439575195, 20.452974319458008, 43.66546630859375, -27.402551651000977, 38.36860656738281, -16.492910385131836, 9.880439758300781, 14.102206230163574, 18.201818466186523, -38.99924087524414, 17.12923812866211, -24.916648864746094, -6.03013277053833, -36.41779708862305, 35.04943084716797, -19.993555068969727, 20.528188705444336, 15.748615264892578, 3.7176194190979004, 33.9019775390625, 18.438501358032227, -33.7273063659668, -29.050457000732422, 12.800798416137695, -6.101528167724609, 23.703075408935547, 33.46171188354492, -3.345083236694336, 2.098323345184326, 36.84286880493164, -54.643436431884766, 2.806163787841797, -43.650421142578125, -10.368943214416504, -0.5577491521835327, 36.29051208496094, 17.90442657470703, 1.9155278205871582, 11.534811973571777, -19.542484283447266, -56.84890365600586, 29.038084030151367, 18.749109268188477, -11.453462600708008, -10.13552474975586, 13.340925216674805, 2.102476119995117, -19.072031021118164, 38.787479400634766, -2.811662435531616, 18.122404098510742, -26.786781311035156, 23.01682472229004, -6.12883996963501, 14.236808776855469, -63.783382415771484, -9.474461555480957, 5.1920576095581055, 0.010250985622406006, 8.071663856506348, 2.7386486530303955, -13.939360618591309, -3.0104262828826904, 6.114602088928223, 41.185890197753906, 30.621355056762695, -23.203092575073242, 12.42371940612793, -40.004371643066406, -8.258014678955078, -20.31655502319336, -6.049491882324219, -40.76125717163086, 0.31203967332839966, -0.4276818633079529, 52.95441818237305, 1.913151741027832, -14.69740104675293, 45.11259841918945, 23.81437873840332, 30.16352653503418, -25.79199981689453, 25.405954360961914, -3.674114227294922, 33.81443405151367, -49.1756591796875, -33.105472564697266, -17.099416732788086, 11.973496437072754, -23.857521057128906, -15.148578643798828, -54.276161193847656, 34.939048767089844, 15.020164489746094, 41.049293518066406, 35.77558135986328, -34.71620178222656, -8.166038513183594, 0.7415796518325806, 19.422311782836914, -32.63616180419922, -28.66930389404297, 11.973518371582031, -40.0778694152832, -35.79550552368164, -55.6733283996582, 24.251264572143555, -20.82549285888672, 13.92503547668457, -4.752821922302246, 56.986507415771484, -41.48588180541992, -23.303197860717773, 2.8196957111358643, 1.2918416261672974, -14.901135444641113, 19.34100914001465, -10.099485397338867, 64.5015640258789, 4.551435947418213, 3.0619430541992188, 24.2670955657959, 11.026615142822266, -19.773298263549805, 20.526283264160156, -6.197798728942871, 59.50037384033203, 17.62584114074707, -21.139402389526367, -9.929815292358398, -11.572505950927734, 9.064245223999023, -23.24918556213379, 38.92506408691406, -22.75565528869629, 8.050363540649414, 24.212501525878906, -52.740543365478516, -28.882490158081055, 16.72390365600586, 29.80917739868164, 8.773384094238281, -38.88374328613281, 11.722411155700684, -24.193965911865234, -13.025044441223145, 0.8275939226150513, -76.86808013916016, -25.185791015625, -27.92643928527832, -4.439695358276367, -4.079397678375244, -26.142789840698242, 29.300867080688477, 11.487799644470215, -28.41087532043457, 21.785120010375977, 52.99453353881836, -56.46097946166992, -3.9836103916168213, 59.35084915161133, -19.71170997619629, -10.189559936523438, 31.376911163330078, 5.269734859466553, -32.37049865722656, -8.927361488342285, -33.649776458740234, -26.013675689697266, -10.59546947479248, 12.172479629516602, 27.573875427246094, -38.23714828491211, -30.647348403930664, 24.07289695739746, -3.428276777267456, 32.83844757080078, -42.1566276550293, 3.5166871547698975, -11.101791381835938, 35.09219741821289, -5.222470283508301, 17.34367561340332, 10.626834869384766, -55.6317024230957, -28.068172454833984, -14.296221733093262, 24.261913299560547, -18.332481384277344, 9.509603500366211, -15.14931583404541, 22.221572875976562, -2.423727035522461, 18.7303524017334, 32.30602264404297, -30.896831512451172, 21.42946434020996, 4.548809051513672, -4.185099124908447, -0.056026458740234375, 29.20882225036621, 3.8105196952819824, -18.0714111328125, 0.5501497387886047, -45.28764724731445, -47.76224136352539, 27.604930877685547, -1.0395015478134155, 31.392724990844727, -12.396100044250488, -24.325441360473633, -41.318016052246094, -2.615231990814209, -12.489075660705566, -1.4006564617156982, -32.04132843017578, 36.1888542175293, -48.1647834777832, -2.749940872192383, -24.977928161621094, 28.993616104125977, 36.588436126708984, 13.678240776062012, 38.762306213378906, -26.099220275878906, -19.204301834106445, 49.926876068115234, 1.0892229080200195, 27.434064865112305, -42.529685974121094, -41.0268440246582, 27.720956802368164, 18.070541381835938, 8.798381805419922, -13.44097900390625, 14.361732482910156, -5.864804267883301, 13.538867950439453, -26.36632537841797, 23.98615264892578, 33.995811462402344, -1.369147539138794, -14.2000093460083, 2.180053949356079, 27.763723373413086, -23.753984451293945, 41.20365524291992, 15.261785507202148, -20.367412567138672, 11.111038208007812, 70.9985580444336, -37.60752868652344, 31.060298919677734, 41.2245979309082, 2.0983386039733887, 2.908269166946411, -63.30162048339844, -47.640037536621094, 23.746814727783203, -8.429985046386719, -19.31814956665039, 4.431070327758789, 56.8492431640625, 9.554929733276367, -10.066730499267578, -7.116971492767334, 17.43843650817871, 23.120885848999023, -12.984621047973633, -27.28031349182129, -2.099456548690796, 31.508148193359375, 37.81648254394531, -13.105631828308105, -12.846273422241211, 59.899566650390625, 25.0894775390625, 33.729637145996094, -31.270048141479492, 0.5276697874069214, -50.120296478271484, -4.992414474487305, -10.921896934509277, 8.512591361999512, 22.264070510864258, 48.098751068115234, -70.84877014160156, -43.38188171386719, -51.72090148925781, -45.69737243652344, 32.799678802490234, 51.09502410888672, 25.776643753051758, -39.65468215942383, 3.317331552505493, 33.19504165649414, -66.99021911621094, -53.39180374145508, -29.59776496887207, 0.24875521659851074, -40.01389694213867, -42.60300827026367, 37.14796829223633, -19.332902908325195, 3.9233665466308594, -4.335282325744629, 25.986183166503906, 27.742740631103516, 39.22843933105469, -28.530170440673828, -14.25635051727295, 3.9521970748901367, 55.83354949951172, 6.757253646850586, 21.355661392211914, 34.5621223449707, 28.27595329284668, -11.955646514892578, 24.65444564819336, 40.972206115722656, -29.820493698120117, 23.540727615356445, 29.23659896850586, -15.129714965820312, 14.770798683166504, -2.0968127250671387, -65.55541229248047, 18.37944793701172, 13.435201644897461, 30.357614517211914, -22.53266716003418, -22.301528930664062, 8.007729530334473, -23.239234924316406, -78.82129669189453, 25.221223831176758, -52.042301177978516, 30.725549697875977, 73.26863861083984, 25.54191017150879, 15.801451683044434, -6.045356273651123, -12.339916229248047, -33.14287567138672, -26.732759475708008, -29.78109359741211, -4.01247501373291, -6.020667552947998, 3.4480602741241455, 3.535447835922241, 40.35594177246094, 33.24346160888672, 14.509603500366211, 10.6294527053833, -82.93231201171875, 3.3546040058135986, -30.46657943725586, -18.452669143676758, -32.681556701660156, -16.92589569091797, 18.287742614746094, -2.4492931365966797, -27.918607711791992, -17.619359970092773, -14.009209632873535, -49.177589416503906, -0.32004639506340027, 31.38768768310547, -38.4489631652832, 17.259592056274414, 0.2945237457752228, 52.1064567565918, -32.51656723022461, -17.587785720825195, 18.44959831237793, -1.107973337173462, 33.4997673034668, 54.912559509277344, -29.07528305053711, 39.657466888427734, 5.7948503494262695, 8.931511878967285, -6.523590087890625, 3.404162883758545, 19.38666534423828, -21.05354881286621, -5.925516128540039, 13.992880821228027, -9.247390747070312, -15.111130714416504, 24.210731506347656, 58.44923782348633, 12.450908660888672, 23.989299774169922, 2.6370530128479004, 15.291800498962402, 61.30358123779297, -9.569268226623535, 5.479775428771973, -4.490730285644531, 53.64712905883789, 38.552703857421875, -32.5764045715332, -15.159832954406738, 2.34350848197937, -10.62417221069336, 13.892974853515625, -10.534103393554688, -73.5347671508789, -0.22621548175811768, -35.0554084777832, 7.770881175994873, 33.131290435791016, 35.1026496887207, -42.30148696899414, 29.20507049560547, -14.746575355529785, 20.698375701904297, -0.8080379962921143, 42.66010284423828, -26.475343704223633, 37.57674026489258, 13.899741172790527, 41.95543670654297, -14.192594528198242, -62.35831832885742, -25.53025245666504, -41.914878845214844, 14.43690299987793, 39.47320556640625, 16.62336540222168, -46.64198303222656, 54.77495574951172, -37.44486618041992, 50.817588806152344, 9.033228874206543, 7.117071151733398, -16.90785026550293, -5.664852619171143, 20.884931564331055, -8.136497497558594, -4.354361057281494, -4.982250213623047, 55.42510986328125, 58.678985595703125, -40.803558349609375, -15.482086181640625, 9.658788681030273, 35.92645263671875, 74.64625549316406, -18.931215286254883, -1.453475832939148, -6.96370267868042, 39.303226470947266, -66.60382080078125, -45.83616256713867, 13.516851425170898, 46.132450103759766, 3.923854112625122, 7.842502117156982, 17.589229583740234, 12.613094329833984, -8.203245162963867, -10.029390335083008, 12.974150657653809, 24.90279769897461, 17.201894760131836, -10.62531852722168, 15.123635292053223, 23.358814239501953, -11.378743171691895, 30.019020080566406, 14.168729782104492, -1.1590126752853394, 16.941699981689453, -17.857807159423828, -40.1372184753418, 26.21333122253418, -60.11579895019531, -30.973812103271484, -10.353989601135254, -25.74968719482422, 7.553410053253174, -8.91286849975586, 32.33787155151367, -40.54918670654297, 12.668452262878418, -26.967546463012695, -20.29869270324707, -3.432669162750244, -43.8875617980957, 22.77488136291504, 7.673231601715088, 35.401126861572266, 37.93000411987305, 7.589405059814453, 21.573562622070312, -43.91478729248047, -43.59431838989258, -22.371685028076172, 46.538787841796875, -65.73101043701172, 41.26618194580078, -25.633033752441406, -1.3343006372451782, 24.927316665649414, 36.991180419921875, -17.8485164642334, -20.682273864746094, -25.86111831665039, 85.19982147216797, 22.158424377441406, -55.221771240234375, 8.868997573852539, 40.4517822265625, 20.195009231567383, -37.49154281616211, -60.26774215698242, -0.42702075839042664, 24.030601501464844, -5.6351494789123535, 6.633744716644287, 6.7879719734191895, 4.771824836730957, 24.022777557373047, -65.33759307861328, 11.741904258728027, 22.66474723815918, -21.343149185180664, -12.796281814575195, 57.93681335449219, -12.280978202819824, 37.46584701538086, -11.8180513381958, 21.26126480102539, 65.00159454345703, 24.01011848449707, -6.614504814147949, -10.902390480041504, 44.570770263671875, 17.096586227416992, 20.84113311767578, 74.72824096679688, 56.20856857299805, 7.607235908508301, 22.405990600585938, 7.219717979431152, 18.47610855102539, 39.53815460205078, -12.229720115661621, 24.48888397216797, 18.89470863342285, 3.5586676597595215, 23.691835403442383, 46.173885345458984, 36.719364166259766, -25.33915901184082, -0.637790322303772, -1.0488109588623047, -3.8227107524871826, -13.874655723571777, -53.5955696105957, 6.785762786865234, -1.0040329694747925, -17.576967239379883, 9.654402732849121, 12.887869834899902, -4.308750629425049, 64.42069244384766, 2.3952817916870117, 11.125125885009766, -9.281362533569336, -4.015666961669922, -34.19047927856445, -1.2045421600341797, -63.63227462768555, 74.2544174194336, 20.086589813232422, -14.9625883102417, -22.583539962768555, -10.1032075881958, -10.189950942993164, 6.653076648712158, -12.994383811950684, -31.257062911987305, -17.612730026245117, 37.66766357421875, 36.3912353515625, 2.9029746055603027, -36.81284713745117, -16.046188354492188, -17.806686401367188, -16.656314849853516, 5.587466239929199, 6.111844062805176, 3.983013868331909, 0.06392073631286621, 17.58608055114746, 22.829158782958984, -18.410642623901367, -27.216018676757812, 17.48070526123047, 33.57393264770508, 13.616223335266113, -0.7446105480194092, 27.286460876464844, -10.596242904663086, 40.130916595458984, 9.758543014526367, -10.078669548034668, -37.421051025390625, -76.21959686279297, 3.709151029586792, -6.306494235992432, 28.15092658996582, 13.686885833740234, 28.441667556762695, 14.142380714416504, 47.40026092529297, -21.814064025878906, -24.178972244262695, 6.974556922912598, -11.306464195251465, -44.65996551513672, -1.9373316764831543, -34.11043930053711, 0.784042239189148, 46.19479751586914, -37.65972137451172, 27.724300384521484, 17.715099334716797, -26.36111068725586, -15.367622375488281, -38.835227966308594, 9.965888977050781, 17.538835525512695, -30.738258361816406, 15.969462394714355, 4.529975891113281, 30.028818130493164, 6.859432697296143, -18.399415969848633, -12.227398872375488, 11.747633934020996, -9.340441703796387, 1.4578752517700195, -24.165647506713867, 25.11744499206543, 6.1706862449646, 36.902610778808594, -21.808162689208984, -28.801057815551758, 35.723690032958984, -4.577347755432129, -9.719892501831055, 12.522968292236328, -4.323265075683594, 33.867279052734375, 58.466331481933594, 8.228729248046875, -26.066513061523438, 5.417348861694336, 14.184228897094727, -24.214982986450195, 11.37421989440918, 61.73435974121094, 36.62442398071289, 1.4662017822265625, -1.7660123109817505, -12.752577781677246, -43.412418365478516, -35.61137008666992, 19.669519424438477, -7.14963436126709, 44.48873519897461, 14.571545600891113, -11.788664817810059, -4.641850471496582, -11.01900863647461, -45.69807052612305, 15.26018238067627, -1.4603509902954102, 60.31592559814453, -66.32098388671875, -22.48333740234375, 32.259429931640625, -8.115443229675293, 0.7660133242607117, 32.6802864074707, -12.267446517944336, 29.346590042114258, 19.770721435546875, 35.41227340698242, -45.68206024169922, 33.51483917236328, -6.303920745849609, 1.8458442687988281, -32.3263053894043, 7.485558032989502, 18.622894287109375, 40.821590423583984, -35.3942756652832, 37.35069274902344, -3.7107434272766113, 5.366068363189697, -56.41966247558594, 37.87085723876953, -26.693267822265625, -2.9559056758880615, 7.389542579650879, -4.232322692871094, -40.028865814208984, 27.044309616088867, -32.121185302734375, -5.953717231750488, -66.67655181884766, 6.601438045501709, 46.67774963378906, 20.289979934692383, -10.121825218200684, -32.187774658203125, -7.77455997467041, 4.169046878814697, -36.4680290222168, -55.28709411621094, -32.0523796081543, 59.785518646240234, 34.30096435546875, 26.656740188598633, 14.823169708251953, -18.545398712158203, -12.710638999938965, 31.901735305786133, -23.762605667114258, 8.256916999816895, 80.45552062988281, 5.873025417327881, 35.02000045776367, 20.297056198120117, 47.46373748779297, -18.619417190551758, 10.041394233703613, -69.2886734008789, -2.0337107181549072, -0.8371080160140991, 4.231317520141602, 1.19813871383667, 14.656001091003418, 13.483586311340332, 98.15190124511719, -34.81245040893555, 18.34748077392578, -37.90850067138672, -14.692304611206055, 19.062423706054688, 2.3655219078063965, 9.107645988464355, 12.529375076293945, 8.37414836883545, -31.413434982299805, -1.9480587244033813, 62.72922897338867, 19.390230178833008, -42.51484680175781, 44.01163864135742, 38.1317024230957, 5.319401264190674, 7.643678188323975, 16.29119110107422, 24.209444046020508, 31.958860397338867, 31.01478385925293, -39.920406341552734, -28.32563591003418, 53.94073486328125, 30.026065826416016, -21.761159896850586, 57.28543472290039, 40.30363464355469, 4.846233367919922, 31.729522705078125, 34.15161895751953, -14.238717079162598, 33.271644592285156, -6.907123565673828, 17.194416046142578, 28.86664390563965, 35.39892578125, -12.193265914916992, 14.093342781066895, 26.475299835205078, -41.53235626220703, 33.541744232177734, 55.70602798461914, -28.931907653808594, 42.248538970947266, 43.21080780029297, -28.409439086914062, 1.207545280456543, -59.60797882080078, -15.69074821472168, -43.93297576904297, 58.862876892089844, -24.579545974731445, -25.081676483154297, 14.966556549072266, 39.56570053100586, -59.56052017211914, -10.089743614196777, 28.890981674194336, 6.6517462730407715, 11.058039665222168, 16.856395721435547, 1.2581360340118408, 57.09441375732422, 18.61945152282715, -12.05690860748291, -68.2722396850586, -7.167295455932617, 10.912201881408691, -39.53363800048828, -14.103559494018555, -42.06800079345703, 41.99741744995117, -38.38685989379883, 41.97087097167969, -11.049875259399414, 30.21178436279297, 11.545011520385742, -19.826154708862305, 16.642793655395508, 1.7493153810501099, -15.944275856018066, -21.06878662109375, -32.52162170410156, -46.31966018676758, -25.230443954467773, -12.830044746398926, -14.310175895690918, -43.13362121582031, 1.6372458934783936, 2.297337293624878, -42.149253845214844, -14.637999534606934, 8.365805625915527, -8.999580383300781, 7.805809020996094, -13.180976867675781, -16.735166549682617, -49.18326187133789, 0.20015257596969604, -0.6452810168266296, -11.973891258239746, 56.198787689208984, 56.095855712890625, -45.26316833496094, -0.595967710018158, -28.1925106048584, -43.536277770996094, 59.80896759033203, -33.25563049316406, 39.30170440673828, 18.735794067382812, 17.069931030273438, -14.56092357635498, -19.999914169311523, 19.19434356689453, 12.849437713623047, -19.30453872680664, 4.486077308654785, -0.7673184871673584, -10.122784614562988, -40.67357635498047, -8.315335273742676, -4.855543613433838, -12.736595153808594, 13.27098560333252, -13.983953475952148, -5.067558288574219, 27.937870025634766, -27.734161376953125, -11.321611404418945, -32.35922622680664, 7.426139831542969, 23.016721725463867, -34.862571716308594, 7.13659143447876, 5.9856276512146, -29.582597732543945, -13.135034561157227, -81.7841796875, 77.0897216796875, 14.819652557373047, 41.42805862426758, -6.336184024810791, -6.203317165374756, 38.14356231689453, -25.225908279418945, 21.954113006591797, -6.783375263214111, 5.333402156829834, 3.462057590484619, -1.406209945678711, -8.68696403503418, 80.97430419921875, 35.089691162109375, 57.84077835083008, 13.034807205200195, 1.5975309610366821, -54.02116012573242, -3.312746286392212, -23.693347930908203, -1.9713385105133057, -12.18554973602295, 70.22804260253906, -41.433963775634766, 4.499236583709717, -33.76531982421875, -30.221744537353516, -23.94968032836914, 11.987491607666016, 9.13906478881836, -44.29780960083008, 41.76560974121094, -13.679669380187988, -8.700211524963379, -23.688697814941406, -2.9560320377349854, 14.839293479919434, 55.3878173828125, -29.922983169555664, 78.46324920654297, -2.4515974521636963, -48.58759689331055, 1.801542043685913, 11.04200267791748, -59.34403991699219, -9.04948902130127, 36.6494140625, 13.351283073425293, 20.797983169555664, 17.22345733642578, -8.106457710266113, -0.8824059367179871, -32.91257095336914, 27.043441772460938, -79.1881332397461, 0.5168605446815491, -29.072071075439453, -33.3592529296875, -1.5550044775009155, 17.367813110351562, -38.453495025634766, 47.13825607299805, 4.597011089324951, -5.839176654815674, -28.795289993286133, -26.18157958984375, 46.6464729309082, -31.634117126464844, -11.948034286499023, -25.435529708862305, 30.115447998046875, 10.008866310119629, -12.726222038269043, -11.861249923706055, 50.334014892578125, 3.7572898864746094, -37.652381896972656, -24.233036041259766, -11.730521202087402, -53.51479721069336, 28.30511474609375, 18.536760330200195, -21.508296966552734, -14.603704452514648, -7.432819366455078, -13.712465286254883, 50.09484100341797, -12.673523902893066, 6.182314395904541, 24.329605102539062, 17.133146286010742, 38.42302322387695, -18.161178588867188, 11.549189567565918, -28.18193244934082, 35.04399871826172, -22.13262176513672, 42.38966369628906, -24.203136444091797, -17.09673500061035, -17.779115676879883, -36.49773406982422, 67.31842041015625, -9.660300254821777, -23.034656524658203, -35.760128021240234, -9.589963912963867, 48.88949203491211, 48.81565856933594, 8.201108932495117, 32.22665786743164, -3.9000022411346436, 14.395965576171875, 1.2210012674331665, 21.124372482299805, 53.52949142456055, -31.94530487060547, 27.624731063842773, 12.112449645996094, -5.903042793273926, -7.399843692779541, 43.29774856567383, 28.422624588012695, 37.444828033447266, 14.669058799743652, -38.07777404785156, 13.677484512329102, -24.04031753540039, 10.161917686462402, -6.409078121185303, 57.51686477661133, 11.342766761779785, -36.718544006347656, -2.1510438919067383, -1.8136048316955566, 43.66917419433594, 2.3750596046447754, -17.25217628479004, 25.761844635009766, 48.48505783081055, 0.5695134401321411, 24.463375091552734, -31.52359390258789, 4.033156394958496, -38.82085418701172, -13.196746826171875, 24.474103927612305, 10.00220775604248, 13.63984489440918, -4.339192867279053, -38.75957107543945, -24.296524047851562, 15.887994766235352, 4.403937816619873, 32.53727722167969, -32.21839141845703, -25.873865127563477, -17.144685745239258, 12.486205101013184, 32.914634704589844, 34.04150390625, 1.218000054359436, -8.373035430908203, 24.831594467163086, 11.352458953857422, 46.72813415527344, -25.161479949951172, 4.949763774871826, 43.997344970703125, 7.229922294616699, -8.820978164672852, -25.490726470947266, 1.0214450359344482, 6.461334705352783, -20.664386749267578, 21.971759796142578, 10.642377853393555, -49.13637924194336, -19.771133422851562, -48.39537811279297, -24.075008392333984, -3.2559103965759277, -31.509735107421875, 11.097293853759766, -89.40252685546875, -26.375734329223633, 37.04034423828125, -9.264002799987793, -20.574588775634766, -10.747920036315918, -5.006643772125244, -10.999184608459473, -16.804452896118164, -53.56642150878906, -32.27877426147461, -39.718292236328125, -16.18189239501953, 19.240171432495117, 19.760894775390625, 6.768243789672852, -23.067224502563477, -5.562687397003174, -14.21127700805664, 5.416920185089111, 9.806564331054688, 24.00992774963379, -6.539654731750488, 8.380955696105957, 47.14640426635742, 17.35433578491211, -82.30758666992188, 77.54383087158203, 47.37284469604492, -5.458952903747559, -59.62266540527344, -31.560731887817383, 16.195751190185547, -0.8194120526313782, 22.447952270507812, 31.137020111083984, 5.474823474884033, -32.048954010009766, 0.6362947225570679, -16.279977798461914, 26.17671012878418, -17.477048873901367, 22.410646438598633, 57.702880859375, 1.1372774839401245, 56.764564514160156, 32.7630729675293, -26.254392623901367, -28.269519805908203, 10.743943214416504, -10.782665252685547, 20.26336097717285, -47.74958419799805, 31.57232666015625, -14.929933547973633, -2.3225650787353516, 13.896977424621582, 11.024273872375488, -40.6840934753418, 46.26554489135742, 30.269296646118164, 27.956361770629883, -13.640750885009766, 30.597984313964844, 23.4333438873291, 23.786394119262695, 29.174468994140625, -9.558320999145508, 21.580551147460938, -36.443382263183594, -61.86452102661133, -62.29595184326172, -4.947988986968994, -24.159931182861328, -4.483503341674805, 3.7164831161499023, -16.345172882080078, 28.201391220092773, 8.38716983795166, 9.134421348571777, 54.87928771972656, 13.113229751586914, -54.973819732666016, -42.81987762451172, -23.360179901123047, -24.498748779296875, -22.610706329345703, 23.700382232666016, -14.468454360961914, -1.4416872262954712, -14.632832527160645, 54.41619110107422, 12.984017372131348, -16.17391014099121, -5.983179569244385, -40.853271484375, -44.45694351196289, -14.551250457763672, 16.54274559020996, 7.512219429016113, 45.61619186401367, 20.40784454345703, 11.633842468261719, -33.64704513549805, -39.42752456665039, 9.052868843078613, 15.99890422821045, -4.711899280548096, -25.25119400024414, 46.28825378417969, 6.340536594390869, 10.367928504943848, 56.61553955078125, 22.568405151367188, 14.03691577911377, 24.93076515197754, -30.03199005126953, 23.89693260192871, 8.976778984069824, -15.055020332336426, -28.40330696105957, -19.182432174682617, -23.344297409057617, -25.317031860351562, -68.85657501220703, 46.46254348754883, -8.652632713317871, -13.603108406066895, -1.9230902194976807, 24.03953742980957, 5.416436195373535, 37.72187042236328, -19.731122970581055, -2.7043697834014893, -23.888446807861328, 51.4715690612793, -18.47793197631836, 49.62863540649414, 32.62797546386719, 12.976089477539062, 21.267475128173828, 47.55164337158203, 26.84032440185547, -56.82622528076172, 7.462892532348633, -56.43618392944336, -8.281006813049316, -0.2412356734275818, 7.849688529968262, 11.424443244934082, -18.002206802368164, 1.1688692569732666, 27.180221557617188, 60.94852066040039, -63.49160385131836, 4.605178356170654, -10.884827613830566, -13.097809791564941, 68.96005249023438, -54.72908020019531, 66.92623138427734, 2.879497528076172, 29.461294174194336, 12.390767097473145, 47.972225189208984, 11.734519004821777, 9.732192993164062, 3.0628857612609863, -29.378503799438477, -7.998941898345947, -1.729238510131836, -30.86050033569336, 36.48482131958008, 64.1155776977539, 19.44562530517578, 37.93644714355469, 13.401355743408203, 43.87533950805664, -33.357791900634766, 24.088024139404297, -19.178770065307617, -13.865317344665527, 12.869681358337402, -27.10611343383789, -17.20001792907715, -36.503211975097656, 10.161316871643066, 12.491394996643066, 1.064164638519287, 5.655804634094238, 8.666593551635742, 46.827232360839844, -2.8445472717285156, 0.8572520017623901, 42.81081771850586, 39.63502883911133, -66.70305633544922, 7.241554260253906, -16.89637565612793, -9.528129577636719, -46.83578872680664, 17.419418334960938, -56.51611328125, -22.37508773803711, -19.784669876098633, 1.3683626651763916, -34.198585510253906, 6.490274906158447, -61.03630828857422, 35.67356491088867, -20.042028427124023, 45.98979949951172, -21.707582473754883, 23.17178726196289, -16.52705192565918, -21.772327423095703, 8.165278434753418, 7.9390668869018555, -5.885419845581055, 22.599822998046875, 12.001911163330078, -17.88034439086914, -46.85017013549805, -11.109952926635742, -16.702054977416992, 5.495697021484375, 9.840235710144043, -19.1364803314209, -18.404455184936523, -15.30150318145752, 12.086870193481445, -16.446849822998047, 16.350688934326172, -3.6775779724121094, -22.669048309326172, -23.794103622436523, -36.48318099975586, 46.73772048950195, 13.38005542755127, -12.261527061462402, 31.60296058654785, 49.93571472167969, -31.824726104736328, 20.313827514648438, -20.768787384033203, -52.24093246459961, -71.26679229736328, 4.548371315002441, -31.792264938354492, -21.9367618560791, -21.45631980895996, 8.430312156677246, -15.272872924804688, -25.190847396850586, 29.16922378540039, -12.972847938537598, -25.18168830871582, -4.14190149307251, -18.067636489868164, -20.720949172973633, -28.668493270874023, -0.5302078723907471, -46.70664978027344, -20.673128128051758, 32.85301971435547, -6.38307523727417, 32.7569694519043, -42.739418029785156, 10.429948806762695, 25.102970123291016, 52.38075637817383, -40.91791534423828, -27.286108016967773, -35.891204833984375, -24.836143493652344, 13.598978996276855, -4.8407158851623535, -29.51056671142578, 58.05994415283203, 39.272037506103516, 9.767948150634766, -39.40393829345703, 13.056966781616211, 30.590675354003906, -14.13443660736084, -44.2625846862793, -48.969398498535156, -21.44685935974121, -49.12324905395508, 21.9312744140625, -14.603714942932129, -7.529695510864258, -0.08546276390552521, -0.5552427172660828, 1.1460292339324951, 53.99849319458008, -18.948060989379883, 4.214462757110596, -13.359932899475098, -24.378864288330078, -45.035160064697266, -29.05543327331543, -0.28238391876220703, 9.032547950744629, -39.84888458251953, -30.03377914428711, -9.527929306030273, 30.322866439819336, -15.076882362365723, 4.520416736602783, 20.742704391479492, -12.290079116821289, -44.74387741088867, 4.951735496520996, 20.846233367919922, 16.146095275878906, -32.927486419677734, 9.185885429382324, 2.0378928184509277, -8.433804512023926, 21.553564071655273, -4.189541339874268, 42.2333984375, 18.72907257080078, -26.844085693359375, -4.294848918914795, -29.56697654724121, 1.0233007669448853, -12.918777465820312, 25.982566833496094, -33.24125289916992, 23.56309700012207, 31.03971290588379, -1.5872156620025635, -41.94028854370117, 3.0442492961883545, -12.95244312286377, 6.012580871582031, -1.5429103374481201, -10.797894477844238, -1.3182718753814697, -12.957719802856445, 32.48125076293945, 0.43082308769226074, -51.72216033935547, -21.264652252197266, 15.933629035949707, 35.626346588134766, 38.894290924072266, -55.68721008300781, -14.861961364746094, 26.13480567932129, -27.29983139038086, -18.725919723510742, 8.315515518188477, -6.353996276855469, 27.247528076171875, -12.439164161682129, 40.635459899902344, 5.049192428588867, -42.59431076049805, -1.2633024454116821, -50.184696197509766, -6.090724468231201, -4.242170333862305, 16.110950469970703, 11.109496116638184, -6.612894058227539, -9.883634567260742, 62.56244659423828, 32.97685241699219, -41.84915542602539, -0.2396242618560791, -16.124832153320312, 43.91401290893555, -4.047722816467285, 11.180914878845215, 9.900156021118164, 57.278934478759766, 44.79961395263672, 13.777313232421875, -60.46134948730469, -0.43651244044303894, 34.37154006958008, 13.727193832397461, 3.9872634410858154, 16.081771850585938, 50.34080505371094, -20.583927154541016, 39.8507194519043, -57.691123962402344, -16.62094497680664, -17.156293869018555, 2.4781603813171387, -24.42912483215332, 9.736092567443848, 2.2322099208831787, -20.245380401611328, 22.327898025512695, 25.001996994018555, 5.608931541442871, 26.726755142211914, -27.39084815979004, 15.795856475830078, 26.06688690185547, -10.201173782348633, -17.14628791809082, 20.569490432739258, 55.488346099853516, 3.3585619926452637, 27.285160064697266, 38.196475982666016, 28.08695411682129, -23.915964126586914, 41.313507080078125, -7.69615364074707, -4.290796756744385, 24.309568405151367, -0.6443668603897095, 39.69397735595703, -1.6357026100158691, -38.800132751464844, -18.201892852783203, 27.830244064331055, -17.215913772583008, 21.068265914916992, -33.98868179321289, 15.892427444458008, 5.019681930541992, -12.289145469665527, -26.264617919921875, 19.661529541015625, -26.2795352935791, 33.15214157104492, -13.305002212524414, 44.22513198852539, -15.80568790435791, 11.439680099487305, 71.03356170654297, 0.7466791272163391, 22.917531967163086, 62.174560546875, -12.371770858764648, -9.398003578186035, -39.654544830322266, -12.736543655395508, -58.3100700378418, 2.529202938079834, 0.5816002488136292, 1.2892402410507202, -40.92878341674805, -11.926145553588867, 19.610027313232422, 6.297680854797363, 17.00160789489746, 0.3609418272972107, -53.84647750854492, -12.98206615447998, -44.8807487487793, 5.112985610961914, -22.17720603942871, 3.7467379570007324, 4.720664024353027, -10.427530288696289, 39.146236419677734, 29.139568328857422, -41.58073425292969, 9.343804359436035, -21.067752838134766, -0.47095057368278503, 18.119094848632812, 6.419814109802246, -10.02230167388916, 39.42259979248047, -17.612897872924805, -32.30997848510742, 24.842952728271484, 40.778141021728516, 35.764549255371094, 50.15267562866211, -46.3972053527832, -21.31633949279785, -58.003021240234375, -60.02069091796875, -61.480342864990234, -33.989280700683594, -9.969613075256348, 36.589019775390625, -27.93868064880371, 1.0572608709335327, -37.40604782104492, -3.6970345973968506, 3.873222827911377, -9.940040588378906, -4.901965618133545, -5.051516056060791, -38.56219482421875, -26.223215103149414, -22.496557235717773, 18.09279441833496, 16.420406341552734, -37.769432067871094, 71.32872009277344, -4.704132080078125, 2.983590602874756, -18.960142135620117, 27.08798599243164, -45.310752868652344, -52.737937927246094, -25.62746810913086, -73.32553100585938, 23.257963180541992, 53.890289306640625, 10.835147857666016, 32.7783317565918, 29.70949363708496, -58.37872314453125, 7.869398593902588, -11.135960578918457, 18.113086700439453, 25.242727279663086, 11.239046096801758, -0.30607134103775024, 19.655811309814453, -3.9255027770996094, 18.372865676879883, -6.988000392913818, 30.02178192138672, 61.49611282348633, 14.04015064239502, -0.8104254007339478, 30.099172592163086, 44.745445251464844, -104.15267181396484, -12.22680377960205, -2.438398838043213, -18.417579650878906, 14.077134132385254, -60.950496673583984, 6.315735340118408, -17.357885360717773, -33.39626693725586, 42.879695892333984, -28.49627685546875, 0.9062885046005249, -20.651752471923828, 11.860103607177734, -5.005675315856934, 10.404102325439453, -4.8359575271606445, 45.31532287597656, -5.617036819458008, -22.033193588256836, -69.15322875976562, -18.660173416137695, -23.73716163635254, -0.45924240350723267, 6.680884838104248, -5.372955322265625, 55.594146728515625, 2.6039350032806396, 30.78559684753418, 45.005401611328125, -35.10490417480469, -28.43193244934082, -8.529924392700195, -37.97150802612305, -1.2578811645507812, 4.312836647033691, -9.282966613769531, 64.2533950805664, 22.17380142211914, -0.3686031103134155, 11.080991744995117, 22.305635452270508, -47.470985412597656, -7.104061603546143, 30.06343650817871, 12.215291023254395, 45.018760681152344, -31.709556579589844, -30.152019500732422, 7.2838358879089355, 28.408832550048828, -7.929421424865723, -0.5156540274620056, -43.04194259643555, -3.7017557621002197, 36.83671569824219, -19.55958366394043, 19.554244995117188, -1.7679417133331299, -7.15795373916626, -17.176939010620117, 33.95995330810547, 0.02693292498588562, -44.90685272216797, 0.3353874087333679, 44.278934478759766, 5.869344711303711, 25.573532104492188, 0.774634599685669, -20.79339027404785, -45.20526885986328, -40.279296875, 26.831863403320312, -17.184593200683594, -36.68669128417969, -19.7821044921875, 7.310327529907227, -30.58218765258789, 0.7303626537322998, 56.56937026977539, -3.485886335372925, -15.145529747009277, 28.42233657836914, -37.10542297363281, -58.97814178466797, -11.036662101745605, -45.933990478515625, 10.364337921142578, 80.04926300048828, -5.452938556671143, 10.574231147766113, -40.94657516479492, -14.856884002685547, -8.18208122253418, 16.84572982788086, 6.196292877197266, -15.278618812561035, 45.96420669555664, -12.261804580688477, -30.10074806213379, -32.592586517333984, 39.713768005371094, 29.213655471801758, 8.956692695617676, 24.21600914001465, -14.469545364379883, 83.51952362060547, 31.6267032623291, 16.074993133544922, 47.148799896240234, 3.876276969909668, -1.3281220197677612, -8.824627876281738, 30.28090476989746, -49.039878845214844, 10.995245933532715, 31.57655143737793, -78.79154205322266, 31.61722755432129, 12.10722827911377, 17.244850158691406, 29.915578842163086, 4.974138259887695, -4.406002998352051, -33.90481185913086, -15.336472511291504, 19.426408767700195, -34.07078552246094, -14.598312377929688, 24.081087112426758, 13.506263732910156, 19.3203125, 23.291906356811523, 29.02894401550293, 13.581605911254883, 3.5389957427978516, 11.09664249420166, -5.059622764587402, -55.34538269042969, -6.7922468185424805, -21.195507049560547, 37.93085861206055, 21.249298095703125, -4.734182834625244, 11.401592254638672, 69.55970001220703, -21.293140411376953, -13.878382682800293, 12.266385078430176, -54.4123420715332, -5.363124847412109, -50.54800796508789, -21.699420928955078, 9.715214729309082, 10.032808303833008, 0.6280403137207031, 33.513343811035156, -23.765777587890625, -14.361863136291504, -4.070104122161865, -51.1773567199707, 6.1930389404296875, -17.224096298217773, -20.999486923217773, 23.83997917175293, 5.834846496582031, -32.08049774169922, -67.6681137084961, -51.25403594970703, 7.20045280456543, 22.689390182495117, -45.6690559387207, 18.443143844604492, -28.799610137939453, -27.39914894104004, -61.038246154785156, -25.34130096435547, -6.112349033355713, 21.437772750854492, -60.47016906738281, 5.969277381896973, -18.540061950683594, -46.9603385925293, -64.28046417236328, -8.899073600769043, -17.208660125732422, -30.33340072631836, -17.08247947692871, -41.23642349243164, 53.40913391113281, 27.212512969970703, -6.188628196716309, 22.91973304748535, 25.30245018005371, 55.15973663330078, 3.3975324630737305, -46.921424865722656, 31.429943084716797, 13.017518997192383, -16.59703254699707, -23.63321304321289, -26.135786056518555, -3.5493767261505127, 19.913612365722656, 20.04474449157715, 23.577856063842773, 27.687246322631836, 23.440967559814453, -28.62059211730957, -18.820714950561523, 15.011927604675293, 11.517350196838379, -44.34154510498047, -77.03778839111328, 4.449569225311279, 7.213016033172607, -5.508466720581055, -5.776037216186523, 20.858661651611328, 29.97785758972168, -30.64797019958496, -36.67123794555664, 4.189390659332275, 66.5765380859375, -2.486194133758545, -17.71054458618164, 26.29891014099121, -19.48624038696289, -21.441421508789062, -68.4696273803711, -8.699882507324219, -28.16152572631836, 54.0929069519043, -18.881441116333008, -16.821460723876953, 51.408714294433594, 9.897549629211426, 26.904842376708984, -46.863677978515625, 26.42389488220215, -39.79762268066406, -26.829936981201172, -23.793697357177734, -5.0095744132995605, -13.16639518737793, 43.32455062866211, -42.530311584472656, -21.61103630065918, -38.38359069824219, 38.48045349121094, -38.476009368896484, 13.080168724060059, -5.322134494781494, 8.720367431640625, -30.743066787719727, -72.06151580810547, -41.46100997924805, 3.254932165145874, 64.4739761352539, -87.08812713623047, -17.940542221069336, -0.536018967628479, 13.092591285705566, -27.54347801208496, -37.07721710205078, 25.158340454101562, 40.48297882080078, -14.079326629638672, -14.895386695861816, 38.87786102294922, -64.82254791259766, 11.479293823242188, -21.694303512573242, -2.3621602058410645, 20.84283447265625, 12.364036560058594, 17.646934509277344, 3.74642276763916, -3.9643542766571045, -40.953948974609375, -1.580592155456543, -10.005144119262695, -32.10188674926758, -18.46160125732422, 26.636926651000977, -48.083778381347656, 7.739863395690918, -29.57267189025879, 8.107666015625, -13.866466522216797, 35.757225036621094, -21.1241397857666, 23.723228454589844, 28.284128189086914, -31.12944984436035, 13.421850204467773, 19.234777450561523, 9.379074096679688, 7.94093656539917, 52.53609848022461, -18.814380645751953, -53.1954460144043, -8.168183326721191, -28.744848251342773, 7.446387767791748, -17.48258399963379, -44.2398681640625, 25.203453063964844, 27.994932174682617, 31.569761276245117, 18.207324981689453, -2.8354310989379883, 3.7199454307556152, -26.016069412231445, 33.10614013671875, -13.64619255065918, 9.28919792175293, 11.558860778808594, -16.789175033569336, 8.188071250915527, 27.703025817871094, 4.495909214019775, 17.353227615356445, 10.332008361816406, -40.75194549560547, -41.3687858581543, 39.2109260559082, -50.947288513183594, -14.165146827697754, -1.0758293867111206, 40.72272872924805, 25.271652221679688, 20.713790893554688, 43.37788009643555, -5.863746166229248, -5.933067321777344, -12.835630416870117, -18.457534790039062, 21.555618286132812, -44.6043701171875, -17.47137451171875, 6.269128322601318, 50.63193130493164, 23.352102279663086, 68.37033081054688, 15.616887092590332, -20.78761100769043, 51.37746810913086, -78.19480895996094, 22.124370574951172, -7.0656561851501465, -39.96887969970703, -13.525534629821777, -0.16454273462295532, -11.393802642822266, -44.099159240722656, 26.885845184326172, -15.036722183227539, -13.0488920211792, -0.05635225772857666, -86.49205780029297, 27.86330795288086, 26.660972595214844, 40.92158889770508, -17.439847946166992, 5.980902671813965, -14.96491527557373, -10.821418762207031, -16.155210494995117, -13.737542152404785, -9.448564529418945, -38.843441009521484, -34.699928283691406, -51.75872802734375, -29.33365249633789, 1.6533384323120117, -63.3353385925293, -1.4237428903579712, 85.6198959350586, 22.957101821899414, 61.72270202636719, 12.883241653442383, 12.977505683898926, 27.222288131713867, -5.427733421325684, 23.527917861938477, -18.578838348388672, 24.751705169677734, 11.332524299621582, 19.317424774169922, -8.200794219970703, -44.366790771484375, -4.545655727386475, 10.691624641418457, -37.2100715637207, -44.16212463378906, 14.10033893585205, 32.248355865478516, 3.513272762298584, -39.3983154296875, -42.22981262207031, -20.297252655029297, -19.039112091064453, -2.3679630756378174, 16.02565574645996, -7.515252590179443, -9.918251991271973, -34.83375549316406, -32.073246002197266, 22.861536026000977, -29.50152015686035, -7.214189529418945, -12.517828941345215, 42.53375244140625, 8.760087013244629, -10.330351829528809, 10.123011589050293, -41.21516418457031, 28.383695602416992, 9.082019805908203, -70.49638366699219, -0.9545342922210693, -38.64594268798828, 6.571619033813477, -15.562384605407715, -8.734814643859863, 47.83740997314453, -42.32490921020508, 36.61433410644531, -60.033836364746094, -38.77140808105469, 27.936683654785156, -28.76688575744629, 23.197338104248047, 4.050296783447266, 2.5360820293426514, 27.495847702026367, -0.1067972183227539, -18.619192123413086, -28.933862686157227, -17.074596405029297, -31.974658966064453, -20.908811569213867, -21.918691635131836, -83.32183837890625, -24.91942024230957, 6.780978202819824, 2.1738169193267822, -41.64826583862305, 54.17807388305664, -23.79389762878418, -38.367008209228516, -29.67789649963379, -40.721553802490234, 30.780410766601562, -0.03473740816116333, -21.626461029052734, -19.34673309326172, 0.6114227771759033, 8.282899856567383, 6.622713088989258, 52.95750045776367, 14.805525779724121, 23.81925392150879, -32.68783950805664, -13.98709774017334, -14.380016326904297, 37.46983337402344, 37.9195556640625, 18.4227352142334, -18.465829849243164, 81.11325073242188, 47.25823974609375, 37.715518951416016, 17.740633010864258, -28.348041534423828, 27.410703659057617, 9.829798698425293, 18.26024055480957, 25.99579620361328, -9.751568794250488, -11.1819429397583, 3.3464910984039307, 5.0652947425842285, -15.678847312927246, 11.390310287475586, -64.35839080810547, 7.475575923919678, 20.34836196899414, 12.684680938720703, -26.203659057617188, 18.802204132080078, -15.753034591674805, -2.1625654697418213, 22.152332305908203, 23.49222183227539, -27.130226135253906, -12.356287956237793, -20.432233810424805, -6.22974967956543, 19.273019790649414, 8.075258255004883, -30.47534942626953, -32.35648727416992, -43.52015686035156, -35.4197883605957, 43.70109558105469, -31.000669479370117, 28.88416290283203, -37.500160217285156, 0.42924022674560547, -3.9409613609313965, 0.42302191257476807, 40.12964630126953, -50.71636199951172, -25.270278930664062, 10.875402450561523, -28.995655059814453, -29.51162338256836, -22.979503631591797, -83.66385650634766, -52.139339447021484, -11.566140174865723, -29.010398864746094, 16.905973434448242, -1.4909533262252808, -14.253646850585938, 10.42197036743164, -5.9083356857299805, -32.02703094482422, -14.632102012634277, -28.307397842407227, -24.640090942382812, -44.58330154418945, -14.323640823364258, 20.11945152282715, -35.63880920410156, 4.360542297363281, 16.872621536254883, 15.079848289489746, -56.67058563232422, -24.386619567871094, -15.483650207519531, -8.742575645446777, -25.3315372467041, 36.65664291381836, 11.563597679138184, 7.011703968048096, 26.407243728637695, -5.0414252281188965, -45.99479675292969, -6.225396156311035, 10.157689094543457, 11.107721328735352, -13.318957328796387, -31.96707534790039, -0.9521584510803223, 63.615169525146484, 5.237308979034424, 15.007465362548828, -56.88166427612305, 10.428367614746094, -49.652618408203125, 68.90371704101562, 4.108135223388672, -86.83493041992188, 63.860877990722656, -26.677139282226562, 35.021766662597656, 13.908021926879883, -24.81802749633789, -24.1506290435791, 16.792604446411133, 21.185773849487305, 1.1749078035354614, -34.59587097167969, 19.66379165649414, 54.845924377441406, 17.23415756225586, 16.095544815063477, 27.623180389404297, 15.195018768310547, 22.10966682434082, 22.759904861450195, -19.901691436767578, 10.790827751159668, -34.79119873046875, 2.640544891357422, 26.455219268798828, 8.075251579284668, -10.595383644104004, 42.98003005981445, -16.79063606262207, -24.52845573425293, 3.937649965286255, -96.74773406982422, 13.621369361877441, 11.359790802001953, 89.9171142578125, -8.816633224487305, -4.970181941986084, 32.68413543701172, 30.457565307617188, 12.256420135498047, -28.282644271850586, 35.152976989746094, 8.503776550292969, -75.1042251586914, 11.280534744262695, -42.53916549682617, 7.124436378479004, -3.991849660873413, 5.086836814880371, -7.122716903686523, -77.9049072265625, -7.596238136291504, 56.977760314941406, 15.240357398986816, 24.222213745117188, 70.61833953857422, 0.18029546737670898, 1.6542997360229492, 1.551538348197937, -26.149593353271484, -26.90538787841797, 52.61179733276367, 29.75491714477539, -28.42486000061035, 20.223861694335938, -4.912065029144287, 5.979971408843994, -33.596248626708984, 44.93103790283203, 10.010465621948242, -42.981197357177734, 32.3928108215332, -34.857200622558594, 27.2705020904541, 13.850801467895508, 1.249793529510498, -16.786081314086914, -18.011699676513672, -32.191009521484375, -66.29759979248047, 14.020318984985352, -26.537797927856445, -16.530994415283203, 53.295623779296875, 37.06583023071289, 52.400577545166016, -21.972335815429688, -14.244097709655762, 6.297126293182373, 20.071868896484375, 25.322002410888672, 33.97133255004883, 8.709078788757324, 7.189478874206543, -26.11224365234375, -33.475868225097656, 14.733196258544922, -15.314484596252441, 43.17832565307617, -23.747621536254883, -25.447086334228516, 60.93186569213867, -21.683984756469727, 2.4425625801086426, 12.950411796569824, 31.38534164428711, 10.502212524414062, 4.051318645477295, -51.79831314086914, 12.514719009399414, -25.30719566345215, -22.42750358581543, -14.341443061828613, -49.15073776245117, -62.10417175292969, 12.953131675720215, -14.770570755004883, -24.814359664916992, 18.21682357788086, -26.007213592529297, 28.592002868652344, 16.142784118652344, -7.514223098754883, 73.35783386230469, -4.608640670776367, 11.940999031066895, -20.92042350769043, -9.45616340637207, 30.681716918945312, 60.289215087890625, 37.60193634033203, 9.33248519897461, -1.8669183254241943, -14.818889617919922, 19.267986297607422, 18.982473373413086, -17.686437606811523, -13.309513092041016, -27.920143127441406, 19.13082504272461, -4.905649185180664, -20.618896484375, -8.059493064880371, 5.072279930114746, 23.547800064086914, -18.028657913208008, 20.653318405151367, -48.448150634765625, -11.626909255981445, -38.46285629272461, -22.370725631713867, -17.43926429748535, 10.984464645385742, -39.05562210083008, -19.865251541137695, 14.605294227600098, 7.313395023345947, -9.854103088378906, 41.93619155883789, -21.185344696044922, -34.80459213256836, -52.38679504394531, 2.0907769203186035, 36.14949417114258, -11.783153533935547, 24.055007934570312, -27.928142547607422, -15.428461074829102, -12.374079704284668, 45.736297607421875, -36.105106353759766, -16.4556884765625, -7.91949462890625, 1.7413691282272339, -51.25447463989258, 3.4539661407470703, 23.234609603881836, -7.354885101318359, -11.553955078125, 17.931684494018555, -48.31291198730469, -18.01163673400879, 22.971406936645508, -7.229117393493652, -25.929859161376953, 46.311519622802734, 9.82298469543457, -4.4398698806762695, -9.427366256713867, -0.334652304649353, -5.215359687805176, -2.5042691230773926, 0.18557989597320557, -33.307403564453125, 38.01188278198242, -23.694791793823242, 3.1096370220184326, -8.560574531555176, 85.80826568603516, 14.583306312561035, 77.33600616455078, -9.764594078063965, -6.726325988769531, 1.3216050863265991, -5.437191009521484, -37.667999267578125, 29.27009391784668, 23.815717697143555, -45.763694763183594, 35.50632858276367, -29.945926666259766, 62.43568420410156, 39.73832702636719, 27.61065673828125, 2.177180290222168, -14.220989227294922, 3.6978955268859863, -21.511699676513672, -40.52659606933594, 24.02663803100586, 23.88103675842285, 4.048401832580566, -10.297962188720703, 18.352514266967773, 24.1890926361084, -23.090177536010742, -9.374005317687988, 50.69521713256836, 18.701929092407227, 9.764126777648926, 18.029827117919922, 47.107078552246094, -29.89834213256836, -9.841726303100586, 19.178773880004883, 11.652433395385742, -7.702493190765381, 14.087974548339844, 11.999961853027344, -32.940956115722656, 10.425046920776367, 6.525688171386719, 45.70454025268555, 47.42108917236328, -13.848962783813477, -88.5908432006836, 20.610126495361328, -2.4777021408081055, -40.23815155029297, 25.29287338256836, 34.673484802246094, -43.2310676574707, 22.756221771240234, -39.440155029296875, -6.391396999359131, 17.53409194946289, -36.63750457763672, -51.68132781982422, 2.0001354217529297, 18.877662658691406, -22.841522216796875, 11.145753860473633, -27.93079948425293, -15.488777160644531, -19.851951599121094, 21.05657196044922, -24.329532623291016, 68.17310333251953, -49.12629699707031, -18.651033401489258, 74.01904296875, 5.596385478973389, -17.45094108581543, -36.30544662475586, 35.22517013549805, 11.278240203857422, -34.439884185791016, -7.2995076179504395, -10.217379570007324, -61.82010269165039, -4.17764949798584, -7.506082534790039, -33.689727783203125, -13.741349220275879, 15.617741584777832, 7.559743881225586, -3.3230953216552734, -29.9625186920166, 0.1656510829925537, 42.86740493774414, -40.95893478393555, -8.182703018188477, 37.39472961425781, -0.3164406418800354, -31.055171966552734, -50.42890930175781, 25.565044403076172, 0.04424586892127991, 17.850414276123047, 10.97877311706543, 3.243577003479004, 3.810105323791504, -31.387344360351562, 28.641334533691406, -29.799062728881836, 21.730243682861328, -10.565943717956543, -6.8904008865356445, -16.96255111694336, -3.2379937171936035, -11.285856246948242, -43.76102828979492, -1.9730637073516846, 5.013559341430664, -12.388672828674316, 41.21677780151367, 18.384185791015625, 3.082000970840454, -1.347163200378418, 6.512189865112305, 11.361684799194336, 22.23567008972168, 25.09463119506836, 42.24710464477539, 3.5690176486968994, 11.967531204223633, 35.53291702270508, 6.757306098937988, 39.33155059814453, 16.320314407348633, -34.59548568725586, -8.052663803100586, 19.151098251342773, 61.10805130004883, 10.693496704101562, -17.8358097076416, -26.512792587280273, -33.992671966552734, -18.26431655883789, 14.809986114501953, 28.734848022460938, -3.812988042831421, 79.49224090576172, -17.608373641967773, 32.56835174560547, 40.997615814208984, -14.226473808288574, -13.642822265625, -7.376955986022949, 12.629773139953613, -23.365570068359375, 66.02826690673828, 6.65034294128418, -27.05988121032715, -18.052404403686523, -30.801002502441406, 11.862958908081055, 23.46916961669922, 10.591034889221191, 61.00923156738281, 4.775343894958496, 25.976116180419922, -43.12286376953125, -25.479154586791992, -19.17386245727539, 9.955606460571289, -22.729637145996094, -1.6599661111831665, 44.53777313232422, -10.459319114685059, -19.184968948364258, 21.55297088623047, 4.581737041473389, 39.84672164916992, -42.26348876953125, 3.1835896968841553, -2.1704485416412354, -62.89945602416992, -23.07040023803711, -11.17442798614502, 18.329999923706055, -21.97812843322754, -6.5024094581604, -0.7412965297698975, 58.77477264404297, -27.205778121948242, 22.42647933959961, -13.827713966369629, 5.245174407958984, 16.367555618286133, -38.17082595825195, -0.8611505031585693, -42.11088562011719, -9.278739929199219, -73.98178100585938, -25.946006774902344, -36.193912506103516, 11.437577247619629, 12.074378967285156, 4.88999080657959, 4.395719051361084, 14.411571502685547, -15.059762001037598, -32.860321044921875, -24.158470153808594, 4.089015007019043, 15.504108428955078, -21.936687469482422, 29.460094451904297, -13.709394454956055, 22.108749389648438, -15.865175247192383, -13.642255783081055, -2.1019392013549805, 3.7251148223876953, 31.657089233398438, 31.134746551513672, 0.7558704018592834, -21.662538528442383, -20.61090660095215, -20.89638900756836, 17.436939239501953, -15.897871971130371, 10.592949867248535, 2.4796180725097656, -1.5681025981903076, -33.807403564453125, -16.469667434692383, 5.034770488739014, 6.192366123199463, -40.65467071533203, -10.893762588500977, -9.35080337524414, 41.23411178588867, 18.001874923706055, 17.864492416381836, -8.329463958740234, 5.068584442138672, -54.49923324584961, 37.42196273803711, -89.40158081054688, -3.0764553546905518, 38.836402893066406, 27.684322357177734, -65.5049057006836, -26.796939849853516, 12.637190818786621, -38.85549545288086, 24.024492263793945, -17.54640007019043, 44.662235260009766, 32.69297790527344, -62.27532196044922, 14.455904006958008, -25.485424041748047, 26.214696884155273, 14.22208023071289, 13.617581367492676, 20.05419158935547, -7.54681921005249, 32.67410659790039, -20.131752014160156, -12.229541778564453, -15.501724243164062, -41.40580368041992, -9.538792610168457, 10.303104400634766, -33.68709945678711, 20.022478103637695, -33.87411880493164, -18.472930908203125, 57.55215835571289, -21.665729522705078, 19.476154327392578, -16.576501846313477, -37.326324462890625, -34.86717987060547, -41.743045806884766, -83.26564025878906, 3.533397674560547, 65.10014343261719, 20.640094757080078, -16.837800979614258, 33.582679748535156, 60.552459716796875, -13.47319507598877, -14.024482727050781, -1.7910099029541016, -51.9158821105957, 14.887564659118652, 55.40778350830078, -18.047208786010742, -33.99711227416992, -6.107361793518066, 41.895660400390625, 41.8121223449707, -16.19940185546875, -42.298702239990234, -45.186981201171875, 10.717214584350586, 52.971256256103516, -43.28927230834961, 13.730542182922363, 46.487159729003906, 10.341096878051758, -25.121685028076172, 5.601916790008545, 15.374063491821289, -10.515719413757324, 67.37098693847656, 32.27971649169922, -7.9285759925842285, -21.738134384155273, 6.284243583679199, -61.86698532104492, -12.451001167297363, 63.30565643310547, 65.8854751586914, 22.45623016357422, -25.617952346801758, 38.289466857910156, -32.72922134399414, -7.93156099319458, -51.8502082824707, 36.29909133911133, -1.7706258296966553, -3.306096076965332, -1.5946320295333862, 2.9755163192749023, 41.51872253417969, 9.521987915039062, -48.721309661865234, -4.714230537414551, 7.20894718170166, -0.7658940553665161, -29.881406784057617, 10.29261302947998, -18.39051055908203, -50.4462890625, -8.053741455078125, 43.743473052978516, 12.054689407348633, 8.51917839050293, 36.26946258544922, 34.94645690917969, -1.2427407503128052, -2.8207902908325195, -7.602357864379883, 6.267457008361816, 30.860109329223633, -54.97137451171875, -63.10615158081055, -1.567069411277771, -18.19548225402832, 8.672930717468262, 27.247533798217773, 22.438283920288086, 8.656466484069824, 11.792672157287598, 27.38381004333496, 23.6319580078125, -16.341169357299805, 35.06527328491211, -24.696022033691406, -6.273902416229248, 13.601383209228516, -9.224486351013184, -14.08199691772461, 3.9153852462768555, -44.00859832763672, 27.001720428466797, -23.847305297851562, 5.368894577026367, -5.037898540496826, 5.4413533210754395, 48.71870040893555, 38.2015495300293, -10.465500831604004, 14.699652671813965, -36.85399627685547, -37.940555572509766, 37.897804260253906, -33.27515411376953, -33.379974365234375, 55.49723815917969, 38.84334182739258, 20.93733024597168, -68.31502532958984, -40.84810256958008, -10.229530334472656, -10.827260971069336, 2.085805892944336, 51.220523834228516, -66.92536926269531, 3.5428342819213867, -12.251590728759766, 28.675107955932617, -39.7556266784668, 22.999242782592773, 3.1339190006256104, 39.80621337890625, 23.934043884277344, 27.88160514831543, -51.24702072143555, -18.000917434692383, 35.63527297973633, 0.6202235221862793, 48.80243682861328, -25.76568603515625, 28.879838943481445, 59.505313873291016, -2.893918037414551, -42.29002380371094, -3.262749195098877, -11.849597930908203, 49.02043151855469, -3.7360453605651855, 16.765628814697266, 75.2210922241211, -25.255903244018555, -30.45102882385254, -17.29965591430664, -34.783050537109375, -30.127290725708008, 13.551533699035645, -22.633663177490234, -83.61344909667969, -26.007587432861328, 4.067347526550293, -3.5314698219299316, -26.63711929321289, 32.66139602661133, -4.878620147705078, -4.144227981567383, -11.380351066589355, 53.798404693603516, -60.477718353271484, -1.731535792350769, -23.246313095092773, 20.648910522460938, 48.951805114746094, -32.93226623535156, 11.268424987792969, 22.605567932128906, -13.372709274291992, 16.71396255493164, 1.6790810823440552, 8.627812385559082, 37.01055908203125, -41.679405212402344, 31.87350082397461, 26.83258056640625, -7.608138084411621, -18.01235008239746, -53.15810775756836, -16.633258819580078, 28.579421997070312, -21.999204635620117, -38.30973815917969, 68.06697845458984, 8.459489822387695, 23.640907287597656, 2.2033591270446777, -25.801774978637695, 10.968565940856934, -8.215338706970215, 0.4681077003479004, 22.68414306640625, -27.33266830444336, 30.47578239440918, -26.722318649291992, -59.45747756958008, 12.977810859680176, 4.965934753417969, 12.045090675354004, -5.155340194702148, -6.220210552215576, -27.525394439697266, -16.750516891479492, -14.544636726379395, 52.21919631958008, 18.18238639831543, 24.387210845947266, 43.81303024291992, -15.259044647216797, -33.27253341674805, 33.247066497802734, 10.236865997314453, -28.928478240966797, 5.331458568572998, -25.09786605834961, 35.94647216796875, 28.656299591064453, 6.592588424682617, 39.5302619934082, -12.784436225891113, 58.1017951965332, -66.46073150634766, 17.968435287475586, -1.6664280891418457, 8.751087188720703, -16.776430130004883, -19.44408416748047, 12.247056007385254, 4.5888471603393555, 58.14972686767578, 44.35853576660156, 29.1568546295166, 2.959968090057373, 11.073404312133789, 4.923798561096191, 6.364240646362305, -25.9680118560791, 15.339262008666992, -38.194889068603516, -35.056243896484375, 48.28651428222656, 16.008501052856445, 9.034361839294434, -19.94234848022461, 5.188718795776367, -4.383219242095947, -22.35956573486328, 27.281007766723633, -35.75421905517578, 31.627960205078125, 25.42931365966797, 79.48165130615234, 37.27314758300781, -3.3035621643066406, -66.3956069946289, -23.677837371826172, -0.21625962853431702, 26.144786834716797, 20.00208282470703, 45.61266326904297, 28.675016403198242, -6.359017848968506, -37.872291564941406, -30.48562240600586, -9.134860038757324, -11.424636840820312, 2.2389891147613525, 11.42457103729248, 48.28116989135742, -7.966711521148682, 52.25651931762695, 26.701025009155273, 12.371920585632324, 19.918838500976562, 1.8162236213684082, -34.019134521484375, -6.149538040161133, -11.761075019836426, -17.63150978088379, 16.536785125732422, 27.060619354248047, 30.936594009399414, 32.79121780395508, 0.5390762090682983, -24.20880699157715, 64.83069610595703, 19.13271713256836, 20.534090042114258, 19.518280029296875, -13.725211143493652, 7.6677069664001465, 40.822601318359375, -12.187724113464355, -39.7240104675293, 5.659383773803711, -3.0506486892700195, 19.08444595336914, -40.540313720703125, -17.9993953704834, -21.178882598876953, -45.26427459716797, -25.213388442993164, -85.85771179199219, -26.913394927978516, -2.021763801574707, -11.664003372192383, 32.91925048828125, -21.248933792114258, -19.31157875061035, 40.87507247924805, -40.717498779296875, 31.291933059692383, 18.223371505737305, 35.080360412597656, 16.14621925354004, -47.44880294799805, -35.658531188964844, 41.160560607910156, -7.134350776672363, 1.7871065139770508, 14.834333419799805, 28.974151611328125, 22.756248474121094, -3.546261787414551, 12.103696823120117, 8.031984329223633, 39.73760986328125, -72.56988525390625, -17.243000030517578, 45.44195556640625, -6.430474281311035, 3.543933153152466, -28.290721893310547, -41.251861572265625, -29.788799285888672, -31.03361701965332, -8.859375953674316, -31.882776260375977, -41.55906295776367, -24.732254028320312, 11.190098762512207, 16.00352668762207, 12.919172286987305, 28.565433502197266, -36.98390579223633, 2.456779718399048, -48.533851623535156, 31.483972549438477, -6.382074356079102, 14.267396926879883, 30.63695526123047, -14.930591583251953, -13.47672176361084, -35.34852600097656, 5.781328201293945, 0.3290064334869385, 21.156482696533203, 3.188878059387207, 6.549983501434326, -12.281683921813965, 23.77916145324707, -21.255714416503906, -44.74735641479492, -66.6500244140625, -16.6309757232666, 11.271350860595703, 39.60708236694336, -32.694889068603516, 27.081607818603516, -8.52072811126709, 23.8606014251709, 31.029375076293945, 63.27605056762695, 8.902850151062012, 9.113249778747559, -5.186740875244141, -23.520320892333984, 7.976598739624023, 28.115478515625, 26.562175750732422, -7.834625244140625, -9.180682182312012, -32.332786560058594, 12.876724243164062, -31.436582565307617, -27.011417388916016, 34.472572326660156, 9.824190139770508, -14.642045021057129, 3.2768237590789795, 62.53830337524414, 25.767484664916992, 63.345855712890625, -6.2697296142578125, -8.474613189697266, -58.81887435913086, -9.806207656860352, 40.75067901611328, 69.97824096679688, 7.192204475402832, 10.319828987121582, -34.38827133178711, 17.10297966003418, 50.68350601196289, 40.14784240722656, -12.907135963439941, -45.66367721557617, 12.046670913696289, -40.861331939697266, -27.82147789001465, 18.698516845703125, -34.560977935791016, -15.161327362060547, 2.772632360458374, -12.398106575012207, -8.485001564025879, 25.972780227661133, -2.4791996479034424, 27.537546157836914, 37.40620422363281, 6.125826835632324, -12.7926025390625, -21.799203872680664, -21.704809188842773, -5.2450995445251465, 30.986433029174805, -16.096675872802734, -62.42879104614258, 7.379169940948486, -0.7346946001052856, -9.222063064575195, 15.15957260131836, 24.466045379638672, -29.490938186645508, 7.294536113739014, -2.1718850135803223, 13.032136917114258, -0.6668597459793091, 1.153993844985962, -67.59423065185547, 41.60881805419922, 29.71861457824707, -4.95876932144165, 5.251441955566406, -91.6953353881836, -1.766162395477295, -33.53358840942383, 20.651599884033203, -27.047882080078125, 27.468345642089844, 7.796281814575195, -22.116100311279297, 19.523649215698242, -20.345346450805664, 23.198694229125977, -28.199146270751953, 17.617591857910156, -6.413085460662842, 24.48375129699707, 40.34878921508789, -63.65903854370117, -0.5008658170700073, -30.7219181060791, 44.541831970214844, -61.349525451660156, 18.29924201965332, -86.24520874023438, 26.50215721130371, 10.219938278198242, -4.218780994415283, 50.56267547607422, -2.0168230533599854, 17.8538761138916, -37.4205207824707, 9.86121940612793, 27.971479415893555, 71.03710174560547, -25.556640625, 0.9177656173706055, 51.65623474121094, 77.93439483642578, 8.407417297363281, 15.631649017333984, -45.11784744262695, 12.119297981262207, -55.84720993041992, -24.134906768798828, 17.468753814697266, -25.903629302978516, -43.29609298706055, 82.64450073242188, -58.239044189453125, 28.645116806030273, 23.335670471191406, -17.304025650024414, 20.43683433532715, -14.316788673400879, -34.720191955566406, -13.35665512084961, -9.482136726379395, 29.871118545532227, 20.669790267944336, -24.351947784423828, 45.92160415649414, -48.539363861083984, -61.196659088134766, -44.68813705444336, -24.213424682617188, -14.689845085144043, -34.5327033996582, -2.5011019706726074, 2.1649744510650635, 43.42020034790039, -23.130321502685547, -0.4610164165496826, -26.150524139404297, 22.494958877563477, -7.930306911468506, 72.50923919677734, -11.911333084106445, 49.847957611083984, -3.763312816619873, -26.95680046081543, 22.27916717529297, 55.350799560546875, -13.057251930236816, 30.4654541015625, 32.14664840698242, -0.36966848373413086, -7.411520481109619, 28.424156188964844, 51.08247375488281, 24.704139709472656, 13.786252975463867, -53.435829162597656, -29.02838706970215, -22.39439582824707, 10.247668266296387, 22.478473663330078, -35.369163513183594, 10.42010498046875, -5.437434196472168, -35.44277572631836, -9.760845184326172, -14.21388053894043, -0.06332612037658691, 57.228721618652344, -1.5009297132492065, -14.007463455200195, 30.99359703063965, 10.443957328796387, 14.96854305267334, 23.365625381469727, 21.099973678588867, 82.00896453857422, -27.367822647094727, 12.237201690673828, 4.539046287536621, -30.841859817504883, -9.68299674987793, 46.05964279174805, 40.42566680908203, -10.838756561279297, 32.324005126953125, -6.512607097625732, 4.117244720458984, -43.87142562866211, 16.53364372253418, -16.91959571838379, -28.493276596069336, 6.247222900390625, -23.69759750366211, 6.186191558837891, -35.47721481323242, -63.304813385009766, -27.154470443725586, 11.94137954711914, 57.52700424194336, -32.940650939941406, -6.56483793258667, -44.47345733642578, 7.246355056762695, 9.674046516418457, -14.185079574584961, 24.754714965820312, 15.89293098449707, -0.5279974937438965, -40.463104248046875, 19.837543487548828, 0.6576802730560303, -37.373355865478516, 2.5383353233337402, -24.13064193725586, -25.932252883911133, -47.41575241088867, 4.338574409484863, 21.591764450073242, -1.1406433582305908, -13.409090995788574, 34.854713439941406, 5.5673956871032715, -35.00997543334961, -39.546749114990234, -64.38803100585938, -59.019535064697266, -13.173127174377441, -39.4692497253418, -8.461395263671875, 8.11607837677002, 37.41606903076172, -25.05963706970215, -1.1758713722229004, 44.676185607910156, -7.818368911743164, 1.738567590713501, 9.703120231628418, -26.024658203125, 12.773818016052246, -32.20105743408203, -29.05133628845215, -26.377267837524414, -12.691760063171387, 16.066112518310547, -30.812068939208984, -29.520456314086914, 24.27005386352539, 15.287846565246582, 0.4555444121360779, 46.48509216308594, 9.124725341796875, 34.757843017578125, 28.488388061523438, 3.1998021602630615, -22.68645477294922, 2.2634570598602295, -29.034399032592773, 32.171974182128906, 26.572717666625977, -4.945280075073242, -68.48048400878906, -0.0923050045967102, 37.42890167236328, -53.71974563598633, 14.211691856384277, 31.126829147338867, 12.025627136230469, -0.2982400059700012, -46.446533203125, 65.02774047851562, -12.0845947265625, -42.44078063964844, 15.0256986618042, -11.713531494140625, 34.25690841674805, 3.6695568561553955, -11.78304386138916, -8.179370880126953, -21.37641143798828, 23.218456268310547, 10.564697265625, 28.435304641723633, -53.36846160888672, 35.98853302001953, -0.5773554444313049, -33.17000961303711, 8.043417930603027, -43.72954559326172, 19.31862449645996, -20.06559181213379, 19.655166625976562, 8.512224197387695, -42.29471206665039, -34.46744918823242, 15.594132423400879, 25.927417755126953, -40.77134323120117, -9.612604141235352, 0.4344311058521271, -41.73523712158203, 26.972808837890625, -41.69993209838867, 0.08667397499084473, -44.167694091796875, -48.65367889404297, -35.49221420288086, 34.11783218383789, -43.27817916870117, -24.413999557495117, 19.450511932373047, -9.455099105834961, 23.99845314025879, -34.588802337646484, 17.220216751098633, -21.065494537353516, 14.588136672973633, -53.63435745239258, -40.687110900878906, -48.58761978149414, 21.18687629699707, 14.150931358337402, 11.156389236450195, -6.978259563446045, 29.87569236755371, -9.017731666564941, -13.76693344116211, 16.099411010742188, 17.094390869140625, -28.805076599121094, -15.877928733825684, 7.68576192855835, 12.998895645141602, -3.5886926651000977, 33.55911636352539, -56.973411560058594, -10.817299842834473, 22.452823638916016, -31.47138786315918, 33.08880615234375, -1.796066403388977, 48.323360443115234, 21.064437866210938, 24.692201614379883, -4.661098480224609, 37.15170669555664, -20.74941635131836, -2.283067226409912, 4.742411136627197, -6.809800148010254, 28.00521469116211, 26.500303268432617, -28.306076049804688, -19.010601043701172, -27.904090881347656, 1.6479740142822266, 6.9372711181640625, 32.71155548095703, -16.67906379699707, 28.15930938720703, 25.51664161682129, 33.515769958496094, 5.240825653076172, 4.1378936767578125, -14.775422096252441, -3.478705406188965, 9.66905689239502, -5.453244686126709, 5.590713024139404, 55.192195892333984, 35.87065124511719, -12.809955596923828, -2.9892005920410156, -20.292028427124023, -8.998695373535156, -1.2048488855361938, 13.447175025939941, 22.303213119506836, 8.898229598999023, 23.835529327392578, -95.53636169433594, -51.52043151855469, -47.73611831665039, 29.94974708557129, -63.888893127441406, -1.9505393505096436, -1.4153850078582764, 7.171196937561035, 5.053384304046631, -9.82941722869873, -20.720991134643555, -68.63580322265625, 90.16397857666016, 6.937836647033691, 7.940840244293213, 28.83701515197754, -11.050293922424316, -52.50828170776367, -22.342498779296875, -36.555240631103516, -2.604504346847534, 28.524311065673828, 15.135127067565918, 23.08768653869629, 61.054710388183594, 14.070697784423828, 7.744170665740967, -35.03144454956055, -18.80966567993164, 44.45692443847656, 30.838008880615234, -12.812347412109375, -8.479676246643066, -33.697505950927734, -32.80481719970703, 16.324195861816406, 3.0374040603637695, -32.30250549316406, -9.525208473205566, -32.40826416015625, 14.937430381774902, -29.119144439697266, 30.329402923583984, 27.1726016998291, -3.5450737476348877, -16.29848861694336, 10.71989631652832, -2.2462663650512695, 26.307769775390625, -11.89757251739502, -16.546689987182617, -24.703176498413086, -27.07684326171875, 7.122655391693115, -45.761837005615234, -41.212703704833984, 24.564170837402344, 14.801302909851074, 47.93772506713867, -2.75642991065979, -41.109397888183594, 59.64285659790039, -6.023392677307129, 21.527986526489258, 19.75647735595703], [-11.899499893188477, -38.69624710083008, -1.546015977859497, 47.766544342041016, 14.269540786743164, 20.026079177856445, -7.286480903625488, -9.265963554382324, -24.073633193969727, -20.239717483520508, -27.95762825012207, 5.156581878662109, -70.43057250976562, -50.84225845336914, 46.55988693237305, -53.49126052856445, 13.780216217041016, -13.011798858642578, 18.925222396850586, -12.273669242858887, -12.780153274536133, 33.58802795410156, 21.690807342529297, -36.54351806640625, -42.35012435913086, -38.0618782043457, -22.942026138305664, 19.007984161376953, -18.11924171447754, 9.866243362426758, -14.781505584716797, 17.328157424926758, -9.575859069824219, -20.013490676879883, -6.239994525909424, -0.13158082962036133, 2.6955270767211914, -2.889190435409546, 16.143339157104492, -20.142494201660156, -8.447673797607422, 16.553443908691406, 38.49187469482422, -1.5641967058181763, -7.0344367027282715, -46.97683334350586, 69.58194732666016, 11.303454399108887, -27.5494384765625, 61.93939208984375, -13.042437553405762, -6.872628211975098, 8.028682708740234, -7.863009929656982, 4.585312843322754, -87.15006256103516, 3.050673246383667, 13.047107696533203, 4.337759017944336, 8.772355079650879, 1.7144112586975098, 23.993328094482422, 0.5302874445915222, 41.71714401245117, 30.939733505249023, -23.72085189819336, 49.59138107299805, 45.33226776123047, -57.4831657409668, 69.2153091430664, -19.39582633972168, 28.60072135925293, -53.0110969543457, 23.11812400817871, 14.863082885742188, 36.71368408203125, 7.677764892578125, 10.545120239257812, -41.236000061035156, 6.625635147094727, -23.96751594543457, 5.432094097137451, -19.507614135742188, -28.81181526184082, 6.733810901641846, -42.496219635009766, 3.7009828090667725, -9.298449516296387, -31.78974723815918, -53.72897720336914, -2.056772470474243, 37.759666442871094, -10.338830947875977, -47.795814514160156, -37.44974136352539, -1.9823837280273438, -9.515981674194336, -13.59425163269043, 50.7611198425293, 42.13053894042969, 2.034213066101074, 46.896385192871094, -42.547428131103516, 27.596715927124023, -0.28784042596817017, -45.05422592163086, -8.7832612991333, -2.634237289428711, -28.813247680664062, -43.69551086425781, 56.512962341308594, 5.765850067138672, 32.159969329833984, -7.464260578155518, 15.189208030700684, 56.36304473876953, -17.603254318237305, -44.031490325927734, -12.682805061340332, -24.82948112487793, 27.94642448425293, -58.13587188720703, 15.013891220092773, 35.039608001708984, 16.910255432128906, 8.08424186706543, -35.83003234863281, -7.593703746795654, 11.227737426757812, 17.7642822265625, 0.9130497574806213, -38.99226379394531, -14.018622398376465, 10.997844696044922, 20.07139778137207, 35.02132034301758, 64.66775512695312, 14.444938659667969, 8.863473892211914, -31.69912338256836, -22.199695587158203, 68.79756164550781, -12.869781494140625, -38.01335906982422, -29.91257095336914, -1.3856974840164185, 26.883651733398438, 11.604199409484863, -5.7205891609191895, -2.1782519817352295, -21.052993774414062, -30.600730895996094, 19.872180938720703, -15.130826950073242, 23.00051498413086, 24.472135543823242, -11.978952407836914, 3.8852357864379883, 20.733352661132812, 43.3177490234375, 0.018283307552337646, -3.9728641510009766, 18.768558502197266, 21.865867614746094, -5.107728004455566, -13.650769233703613, 19.457679748535156, 16.579952239990234, -23.795921325683594, -54.50811767578125, 24.693504333496094, 7.22866678237915, 16.162033081054688, -55.380008697509766, -32.32811737060547, 33.815704345703125, -7.5822272300720215, -26.364789962768555, 14.396732330322266, 3.1947522163391113, 50.72462844848633, 18.162866592407227, 10.449456214904785, -41.26542282104492, 15.464417457580566, 12.525192260742188, 1.573887586593628, -46.6418571472168, 3.6079514026641846, 26.995973587036133, 22.419031143188477, 24.521869659423828, 46.7257194519043, -43.9024772644043, 7.387157440185547, 4.213829517364502, 26.585283279418945, 0.13935089111328125, 18.85394859313965, -40.142578125, 9.939632415771484, -34.71833801269531, 2.9099881649017334, 0.8723373413085938, -30.583984375, 32.231361389160156, -39.08705520629883, -57.487640380859375, 16.278491973876953, -23.459810256958008, -12.301342964172363, 3.2937746047973633, 11.317273139953613, 2.4537177085876465, -41.71644592285156, 33.08406066894531, 37.73749923706055, -22.24562644958496, 16.571531295776367, 68.39173126220703, -6.310296535491943, 2.438188076019287, 26.19485092163086, 39.63832473754883, 25.340370178222656, 19.493566513061523, -35.721099853515625, 1.0637372732162476, 5.913821697235107, -17.15574836730957, -22.146421432495117, 0.32325124740600586, -4.381523609161377, -12.418785095214844, 10.143599510192871, 20.781118392944336, 28.172510147094727, 15.38458251953125, 31.362346649169922, -30.522483825683594, 46.097564697265625, 9.078157424926758, 41.20705032348633, 37.76116943359375, 38.746986389160156, -28.647300720214844, -25.89171028137207, -50.31508255004883, 36.8636474609375, -7.095501899719238, -20.776674270629883, -18.818391799926758, -0.5071494579315186, 3.517322301864624, 42.74834442138672, -33.16349792480469, -31.203243255615234, 3.173759937286377, -32.46181869506836, -2.890838623046875, 18.129621505737305, 1.542367935180664, 26.832073211669922, 7.003998756408691, 7.6282548904418945, 19.34027862548828, 13.048787117004395, -0.7552111744880676, 41.142112731933594, -46.10716247558594, -10.639787673950195, -25.41999626159668, -60.56645202636719, -74.7077865600586, -29.39901351928711, 16.37477684020996, 47.627437591552734, 17.040430068969727, -38.19035339355469, -49.49311447143555, 3.431821346282959, -17.679901123046875, 39.43251419067383, -2.7569026947021484, -32.027225494384766, 52.69462966918945, 17.516878128051758, -41.2343635559082, 16.826824188232422, 15.800573348999023, 30.832752227783203, -12.504302978515625, -0.16478046774864197, 24.120553970336914, -16.225433349609375, -25.15997314453125, -5.175386905670166, 44.854366302490234, 37.04030227661133, 9.657675743103027, -9.876289367675781, -16.862207412719727, 3.339773178100586, -9.785383224487305, 74.98238372802734, 15.103870391845703, -11.619739532470703, 27.632131576538086, 15.208199501037598, -3.869581460952759, -2.509795665740967, -32.63850021362305, 21.088653564453125, 16.573944091796875, 3.6291439533233643, -52.425052642822266, 11.537765502929688, -56.9445686340332, -21.06011962890625, -22.616226196289062, 3.8377912044525146, -53.26927185058594, 15.476484298706055, 28.000913619995117, 19.31590461730957, -25.827329635620117, 4.647846698760986, -63.13755798339844, -54.71616744995117, 68.63644409179688, -33.16054916381836, -5.66257905960083, -15.589977264404297, 25.147315979003906, 21.511497497558594, -6.103673458099365, 38.35413360595703, 56.965660095214844, -1.6561027765274048, 0.38591888546943665, 76.50318908691406, -10.969146728515625, 13.245978355407715, 27.846887588500977, -64.02361297607422, 5.087296485900879, 19.428312301635742, 31.103525161743164, 26.152803421020508, 47.0131721496582, 6.568076133728027, 14.138965606689453, 21.39596939086914, 5.437398433685303, 4.65018892288208, 22.238584518432617, -10.55925464630127, -43.952396392822266, -52.72271728515625, -5.143296241760254, -42.159950256347656, 33.6992073059082, -1.7012933492660522, 30.149301528930664, -4.294334411621094, 54.91261672973633, 43.673858642578125, 24.204015731811523, -8.65998649597168, -45.78546905517578, 43.477420806884766, 29.010780334472656, 22.62066078186035, 32.80205535888672, 3.0295400619506836, 33.01471710205078, -27.132925033569336, 6.3065056800842285, 15.69923210144043, -28.09494972229004, -24.414236068725586, 4.936634063720703, 42.26240539550781, -23.88088035583496, 23.835315704345703, -12.628499031066895, -44.114288330078125, 2.682742118835449, -32.314754486083984, 19.467483520507812, -40.96469497680664, 5.65764045715332, 6.322250843048096, 16.7070255279541, 4.627307891845703, -50.56764602661133, 22.325464248657227, -36.75605392456055, 23.315250396728516, 49.03120803833008, -43.8893928527832, 6.319743633270264, -2.330798625946045, 17.714813232421875, 4.2915520668029785, 28.870290756225586, -12.015069961547852, -9.504792213439941, -35.24228286743164, 16.10472297668457, -70.71585083007812, -4.7353515625, 28.015390396118164, -3.298668384552002, 26.429515838623047, -10.730253219604492, -26.69147300720215, 10.63171672821045, -37.03620147705078, 49.29119110107422, -1.578102469444275, -41.28837203979492, 9.565825462341309, -11.895689010620117, 13.256133079528809, 17.690305709838867, -9.020184516906738, -15.533637046813965, -52.85481643676758, 35.20774459838867, -1.943586826324463, 102.5159683227539, -15.59388256072998, -17.455833435058594, 19.713298797607422, 31.592533111572266, -5.62235164642334, 17.445526123046875, -42.05865478515625, 15.867781639099121, -55.81977081298828, -15.826424598693848, -2.4702260494232178, 5.195149898529053, -13.852450370788574, -40.62409210205078, -18.275821685791016, 22.3216495513916, -35.74684143066406, 33.43008804321289, 4.302783489227295, -21.398643493652344, -22.15536880493164, -18.33414077758789, 26.754558563232422, 40.164756774902344, 45.363319396972656, -15.83840274810791, -50.0838623046875, -54.896793365478516, -4.3283257484436035, -54.154666900634766, -8.956038475036621, -15.722158432006836, 17.229394912719727, -9.787055969238281, 33.846797943115234, 37.994300842285156, -25.806867599487305, -0.23545169830322266, -76.85604858398438, 29.304162979125977, 30.57318115234375, 20.912490844726562, 8.890087127685547, -16.246498107910156, -7.439999103546143, 40.48189163208008, -26.74335289001465, -0.7036614418029785, 17.73388671875, 84.30915832519531, -9.370362281799316, -2.0222952365875244, -2.571903944015503, 19.394466400146484, 9.296152114868164, -55.66841506958008, -6.6136980056762695, -10.210383415222168, 17.954898834228516, 9.30738639831543, -20.77375602722168, 46.7745475769043, -23.233253479003906, 9.111968994140625, -40.64374542236328, 26.850833892822266, -10.472393989562988, -2.2191312313079834, 88.12406158447266, -8.097780227661133, 13.890220642089844, -76.41197967529297, -38.65837478637695, -9.702603340148926, 23.05457305908203, 4.031057357788086, 4.6357316970825195, -8.191543579101562, 12.500566482543945, 7.342708110809326, -10.948190689086914, 23.23531150817871, 53.38928985595703, -33.95609664916992, -15.024054527282715, -52.16667938232422, -29.550264358520508, -8.92250919342041, 49.424503326416016, 15.927903175354004, -5.898322582244873, 4.413675308227539, 3.314967632293701, 47.13327407836914, -1.513648271560669, 12.124372482299805, 20.384870529174805, -18.82281494140625, 4.541802406311035, 28.139129638671875, -5.376957893371582, -44.50039291381836, -33.313804626464844, -61.86318588256836, -15.965989112854004, -5.678476333618164, -14.694165229797363, 20.144115447998047, -22.982769012451172, 59.0432243347168, 9.203453063964844, 23.327495574951172, -12.536711692810059, -45.743045806884766, -38.46864318847656, -49.07204818725586, -24.43556785583496, 0.22063666582107544, -23.462615966796875, 9.809982299804688, 46.99675369262695, -10.691096305847168, -45.385711669921875, 21.046655654907227, 9.361186027526855, 17.948143005371094, -13.34390926361084, -2.500985622406006, 28.05316925048828, 36.89013671875, 43.97810745239258, -2.0809273719787598, -48.40896987915039, -51.65814971923828, 2.2067458629608154, 21.223262786865234, -45.823997497558594, 26.223793029785156, -4.896352291107178, 19.87523078918457, 23.18335723876953, 3.632903575897217, 19.555078506469727, 42.519569396972656, 11.680113792419434, 43.636592864990234, 30.79977798461914, -27.251047134399414, -9.792858123779297, 10.224400520324707, -59.53255081176758, -4.5336809158325195, 26.01700210571289, 24.810346603393555, 23.471817016601562, 74.60868072509766, 18.345252990722656, 23.957365036010742, 29.611364364624023, -5.4078474044799805, 47.36456298828125, 73.59413146972656, -49.468116760253906, 35.867061614990234, 28.569232940673828, -41.30855178833008, 10.31128978729248, 0.39091068506240845, -6.545881748199463, -11.975017547607422, -11.38182544708252, 6.948722839355469, 2.257803440093994, -0.39817243814468384, 6.288093566894531, -8.463032722473145, 96.8674087524414, -53.73236846923828, -12.038792610168457, 5.538782119750977, 45.25389862060547, -36.16737747192383, -44.89469909667969, -22.522592544555664, 1.3415507078170776, 30.235626220703125, 1.5852394104003906, 44.81569290161133, -13.42236042022705, 9.977967262268066, -26.512407302856445, -10.840243339538574, 2.837841510772705, -6.31643533706665, 28.64558219909668, -6.580982685089111, 51.5016975402832, 1.411194920539856, 35.61654281616211, -48.14226531982422, -1.345598578453064, -18.517358779907227, 18.489912033081055, -25.344970703125, -8.262298583984375, 2.530031681060791, -14.859301567077637, 38.9426383972168, -18.50258445739746, 45.08975601196289, 73.87469482421875, -18.579206466674805, -29.612287521362305, -16.116226196289062, 23.448196411132812, -0.9190173745155334, 73.11575317382812, 106.06239318847656, -33.945899963378906, -38.39548110961914, 12.054366111755371, 31.186796188354492, 27.46503257751465, 23.041282653808594, -2.2610671520233154, -13.606124877929688, -31.562700271606445, 5.843753814697266, 31.613271713256836, 2.448089122772217, -32.12831497192383, 36.45262908935547, -15.601902961730957, 26.07189178466797, -21.911100387573242, 16.52155113220215, -17.76030158996582, -25.20845603942871, 26.65769386291504, 24.421428680419922, 19.36164665222168, -26.946678161621094, 36.27659225463867, -39.722206115722656, 3.8862972259521484, 25.174896240234375, -41.340457916259766, 14.529899597167969, -3.866936445236206, -31.635555267333984, 33.862667083740234, -31.668519973754883, 33.09354019165039, 15.2134428024292, -38.63032913208008, -19.48811912536621, 30.38005256652832, -5.513382911682129, 10.087109565734863, -25.912134170532227, 37.205013275146484, -23.56503677368164, 2.216413974761963, -40.07673645019531, 17.625303268432617, 10.90212631225586, 14.99952507019043, -30.591272354125977, 20.0744686126709, -9.264965057373047, -44.979896545410156, -21.03350067138672, 13.161368370056152, 25.31613540649414, 24.88461685180664, 24.692607879638672, -9.40286922454834, -20.26068687438965, 19.938955307006836, -17.360023498535156, 22.30752944946289, 31.338136672973633, 20.39150047302246, -2.7606143951416016, 0.49403268098831177, 19.068435668945312, 15.82206916809082, -8.866132736206055, 38.41755294799805, -23.832326889038086, -67.0094223022461, 45.77437210083008, 25.777944564819336, 11.584185600280762, 12.465214729309082, -27.17247200012207, -28.014297485351562, -30.76327896118164, 15.995184898376465, -27.564075469970703, 0.6464720964431763, -1.5910474061965942, 14.90578556060791, -4.869929313659668, 30.240924835205078, -19.810340881347656, -47.83802032470703, -2.618569850921631, -16.111371994018555, 51.391117095947266, 38.848636627197266, 5.560730457305908, -20.72153663635254, -25.523778915405273, 1.3012194633483887, -39.27943801879883, 56.803985595703125, 25.76616859436035, -39.90913391113281, 4.831951141357422, -12.153851509094238, -40.52080535888672, -7.098649978637695, 36.355323791503906, -12.968917846679688, 30.656057357788086, -48.869354248046875, -1.714369773864746, -49.066734313964844, 31.691638946533203, -3.110619306564331, 11.25781536102295, -4.748239040374756, -1.274787425994873, -36.65436935424805, -27.14443588256836, 21.725749969482422, -36.22090148925781, 72.65998077392578, 31.485286712646484, -3.8903870582580566, -72.14177703857422, 38.8490104675293, -21.469945907592773, 5.608543395996094, -28.8459415435791, 10.725202560424805, -45.56998062133789, -0.8127318024635315, 0.9827932715415955, -7.129055023193359, -5.043681621551514, -4.253462791442871, 8.772140502929688, 37.1122932434082, -22.129560470581055, -10.609047889709473, 20.020681381225586, 10.46255874633789, 1.5331426858901978, -14.94753646850586, -30.757387161254883, 36.25537109375, 33.271915435791016, -26.97838020324707, 14.028371810913086, 16.124570846557617, 10.25803279876709, -9.602640151977539, -17.921598434448242, -11.693191528320312, -0.19129586219787598, -1.4023737907409668, 27.06330108642578, 20.92391586303711, 17.716440200805664, 2.0790772438049316, -26.73400115966797, 12.602594375610352, 3.139435052871704, 30.099777221679688, -1.0979081392288208, 10.271931648254395, 31.0136775970459, -61.768428802490234, -48.4151496887207, -3.1527047157287598, 40.93037033081055, 1.0295100212097168, 37.76705551147461, -28.524433135986328, -1.4287943840026855, -68.30753326416016, 23.790752410888672, 31.079349517822266, -25.40787696838379, -13.423538208007812, -44.323707580566406, 47.89646911621094, 6.3703484535217285, 24.582849502563477, -34.22206497192383, 9.932714462280273, 10.756929397583008, 46.50326156616211, 6.963449001312256, -48.04645919799805, -9.88115406036377, -55.487586975097656, 18.219823837280273, -26.51439666748047, 22.462989807128906, 37.51596450805664, -48.73552322387695, -23.34905242919922, -10.6682767868042, 17.744291305541992, -50.71853256225586, -8.215744972229004, -31.432615280151367, 13.524870872497559, -19.738933563232422, 12.171076774597168, 6.84005880355835, 42.14387893676758, -6.5998430252075195, -42.37987518310547, 28.71915054321289, -16.526123046875, 4.121825695037842, -64.92809295654297, 60.37234878540039, -35.590614318847656, 25.231122970581055, 23.136394500732422, -44.66259002685547, -42.10943603515625, -19.132556915283203, -15.562825202941895, 17.205535888671875, -23.153141021728516, -37.627418518066406, -25.868432998657227, -25.462947845458984, -3.2993640899658203, -30.642051696777344, -47.21727752685547, -12.118857383728027, -12.202672958374023, 40.85955047607422, -20.023685455322266, -1.6565847396850586, 1.1449460983276367, 40.75410079956055, 54.64363479614258, 8.454038619995117, 20.6673526763916, -49.0970573425293, 8.2356538772583, -40.817081451416016, -16.961814880371094, -24.271434783935547, 58.35984802246094, -13.287208557128906, 2.120307683944702, -16.07475471496582, 4.271455764770508, -44.77779006958008, 18.329673767089844, -7.216283321380615, 26.278663635253906, -24.124835968017578, -17.945669174194336, -0.7904307842254639, 9.161797523498535, -13.077078819274902, -29.939632415771484, -22.15472984313965, -8.988518714904785, -27.808252334594727, -24.428993225097656, -10.120878219604492, 60.95652389526367, 9.131914138793945, -9.725852966308594, 0.8504798412322998, -5.494663715362549, 1.9060120582580566, -55.45589828491211, 0.383658766746521, 35.96963119506836, 15.982967376708984, -15.300322532653809, -25.418508529663086, 14.709765434265137, -6.100261211395264, -23.507917404174805, -6.271883487701416, -6.4541754722595215, 93.64081573486328, -28.868982315063477, -44.52886962890625, 31.341218948364258, 12.92542552947998, -6.832834243774414, 16.409502029418945, -17.758047103881836, 1.1779751777648926, -40.12288284301758, -50.52503967285156, 38.47905731201172, 35.06757354736328, 24.15285873413086, 32.070762634277344, -9.614303588867188, -28.507469177246094, 39.14166259765625, 16.418123245239258, -40.83551788330078, 64.39249420166016, -32.388267517089844, 9.646812438964844, -9.105735778808594, -24.99362564086914, -37.58985900878906, 32.653961181640625, 30.814632415771484, -38.71355056762695, 5.618908882141113, 30.988183975219727, 10.50783634185791, -13.325519561767578, 55.022193908691406, -22.00399398803711, 47.623435974121094, 34.564701080322266, 31.32819366455078, -13.538115501403809, 34.98521423339844, -55.52610778808594, 9.906356811523438, -12.02010726928711, 20.156896591186523, -59.44499588012695, -13.405394554138184, -31.755765914916992, -45.98624038696289, -8.69577407836914, 12.955719947814941, 46.70714569091797, 33.80231475830078, 22.576265335083008, -8.474964141845703, 20.064453125, 1.0449012517929077, -13.456847190856934, 19.261322021484375, 15.121707916259766, -28.820009231567383, -41.837825775146484, -44.94853591918945, 31.550737380981445, 11.223787307739258, -14.816179275512695, 1.6133357286453247, -5.993094444274902, -14.514165878295898, -82.93531036376953, -30.73845672607422, -12.851709365844727, -22.78955078125, 19.363555908203125, -2.636979579925537, -40.152099609375, 30.582502365112305, -23.105152130126953, 29.881013870239258, 3.4029760360717773, -8.179350852966309, 3.3095972537994385, 10.883936882019043, -7.899140357971191, -14.454935073852539, 11.690763473510742, -17.85546112060547, 58.3663444519043, -7.555810451507568, -15.065712928771973, -26.592248916625977, 70.91416931152344, -18.497802734375, 10.163802146911621, -44.22900390625, 10.20851993560791, -14.998483657836914, -8.43598461151123, 3.7716212272644043, 31.92402458190918, -15.034902572631836, -9.327306747436523, -11.66543197631836, -3.900001287460327, -54.852989196777344, 51.96673583984375, -56.54667282104492, -27.591402053833008, -16.7344913482666, 41.317787170410156, -18.404659271240234, 17.797876358032227, -7.947307586669922, -11.751764297485352, 7.294826507568359, 12.563911437988281, 17.104841232299805, -16.08521270751953, 10.608858108520508, -33.86070251464844, 26.027734756469727, 5.995671272277832, 4.091966152191162, -5.270571231842041, 21.06388282775879, 6.605003833770752, 31.397504806518555, -0.41559016704559326, -50.456214904785156, 22.856306076049805, -43.3467903137207, 18.279170989990234, -20.93010711669922, -8.294490814208984, -11.584434509277344, -14.444043159484863, -8.849239349365234, -8.75128173828125, -13.28950023651123, -39.783931732177734, 29.494190216064453, 17.53150749206543, 5.932136535644531, 49.569854736328125, -0.5065933465957642, -27.37163543701172, 2.197775363922119, 38.02376937866211, 12.306358337402344, 0.1979597806930542, -29.799232482910156, 27.81442642211914, -64.16146087646484, 25.97335433959961, 44.66350173950195, 23.005020141601562, -30.918466567993164, 14.33637523651123, 11.877004623413086, -10.760844230651855, -35.191444396972656, 8.352799415588379, 46.839637756347656, -28.642126083374023, 0.6161084175109863, -9.652424812316895, 33.9774055480957, -39.15782928466797, 0.05555415153503418, -37.95198440551758, -22.50263214111328, 12.023541450500488, 3.7471892833709717, 25.401981353759766, 29.526897430419922, -17.939266204833984, -44.3139533996582, 24.207639694213867, -31.548871994018555, -13.669301986694336, 29.894624710083008, -12.155173301696777, -8.999845504760742, -19.40675926208496, -47.233642578125, -33.474063873291016, -0.3452787399291992, 17.238250732421875, 14.527498245239258, 5.906187534332275, 19.8011531829834, -7.9264092445373535, -55.10803985595703, 40.36198043823242, -18.562990188598633, -7.296468257904053, -23.583293914794922, 17.70916748046875, 15.407782554626465, -29.361202239990234, 26.25316619873047, 5.289452075958252, 9.665719032287598, 7.342912197113037, 16.8184814453125, -16.772537231445312, 36.47532272338867, 11.55355167388916, 24.86957359313965, 20.08577537536621, 51.98253631591797, -24.50556182861328, -13.772159576416016, -43.5470085144043, -13.014066696166992, -26.46041488647461, -42.63017654418945, 12.006959915161133, 28.217456817626953, 36.75933074951172, -27.03544807434082, 30.573772430419922, -30.559926986694336, -56.92719650268555, 26.591136932373047, -6.992281436920166, -68.6762466430664, 22.3590087890625, -11.484550476074219, 19.04029083251953, -14.80666446685791, -30.92502784729004, -16.211069107055664, -3.282017707824707, -2.091062068939209, 6.004511833190918, 93.16944122314453, 56.921112060546875, 0.7943270206451416, -32.465110778808594, 14.996849060058594, -13.100245475769043, 18.358308792114258, 10.123035430908203, 31.71733283996582, -6.247126579284668, -71.11650085449219, -1.4357638359069824, -31.243072509765625, -4.867819309234619, -8.883869171142578, -19.844310760498047, -55.945587158203125, -22.296133041381836, 15.14619255065918, -20.151697158813477, -26.438758850097656, -11.09079360961914, -13.166915893554688, -44.96160125732422, 19.38330078125, 4.760409832000732, -15.003198623657227, 9.518529891967773, 21.24882698059082, 0.28712350130081177, -3.3283348083496094, 5.017313480377197, 6.663992404937744, 11.50892448425293, 0.4945875406265259, 36.60926818847656, -40.13948059082031, 48.21440505981445, 13.335939407348633, 45.36851501464844, -18.221973419189453, -14.242097854614258, -31.46662139892578, 63.032798767089844, 12.183709144592285, -22.567598342895508, 29.37714195251465, 11.524803161621094, 9.8025484085083, 7.200234889984131, 21.669382095336914, 55.03406524658203, 13.925057411193848, -41.88872528076172, 20.172391891479492, -27.30838966369629, 4.265375137329102, -3.9053869247436523, -13.790630340576172, -23.61187744140625, -31.084096908569336, -1.2420638799667358, 16.826452255249023, 58.5599250793457, -16.36985206604004, -60.78451919555664, -22.751121520996094, -0.5762053728103638, -1.2104012966156006, -99.13177490234375, -13.156432151794434, 46.41307067871094, -65.97943878173828, 6.945448875427246, 45.40494155883789, 51.14963150024414, 24.21166229248047, -16.680583953857422, 35.62655258178711, -79.94522094726562, -7.415620803833008, 31.39061164855957, -11.538029670715332, -5.077653884887695, 33.04943084716797, -16.12580680847168, -55.71964645385742, -19.019207000732422, 0.41789427399635315, 34.371341705322266, 4.973441123962402, -17.825307846069336, 5.11124849319458, -12.826805114746094, -12.7824125289917, -14.991703033447266, 35.357601165771484, 0.010702520608901978, 9.636894226074219, 43.99730682373047, -0.41966086626052856, 1.507974624633789, 29.223894119262695, -22.832923889160156, 27.2902774810791, -9.72780990600586, -29.404375076293945, 15.383102416992188, -1.8309330940246582, -19.63758087158203, -50.3018684387207, 29.456310272216797, -20.3098087310791, 18.79541015625, 13.848994255065918, 4.31231689453125, 10.045817375183105, 15.022911071777344, -18.556041717529297, -24.789384841918945, 34.044761657714844, -46.756351470947266, 22.229928970336914, 10.35201644897461, -15.85202407836914, 47.291446685791016, -31.533540725708008, -15.124369621276855, 16.797012329101562, -27.06178092956543, -13.369824409484863, -55.360740661621094, -10.988128662109375, 32.270790100097656, 36.854278564453125, -26.3564453125, 16.45074462890625, 5.788485050201416, -5.361191272735596, 9.842316627502441, -46.83340072631836, -10.282623291015625, -19.107196807861328, 15.767216682434082, 70.6932144165039, -11.024260520935059, -13.717652320861816, 2.6571667194366455, -14.96670913696289, 0.8462852239608765, -0.49401426315307617, 8.031984329223633, 42.681941986083984, 42.24840545654297, -14.681829452514648, -40.571598052978516, -19.433673858642578, -60.790283203125, 8.832094192504883, 35.015628814697266, -21.354373931884766, -6.586779594421387, 0.6036697626113892, -40.04817581176758, 58.882938385009766, 60.47489547729492, -25.063234329223633, -1.2965835332870483, -7.210639953613281, -34.12098693847656, 45.62812805175781, -60.13529586791992, 21.26102066040039, -6.4501447677612305, -16.438905715942383, 7.93758487701416, 43.12742233276367, 11.86971664428711, -66.00875091552734, -35.535972595214844, 1.7997965812683105, -51.709903717041016, 23.761926651000977, -28.90526008605957, -21.576871871948242, -31.96872329711914, -4.32439661026001, -36.14908981323242, -35.04985046386719, 37.801361083984375, -42.60976028442383, -9.247775077819824, -38.03628921508789, 32.55729675292969, 36.332088470458984, 1.6832902431488037, 28.957324981689453, -1.361767053604126, -11.059734344482422, -32.41867446899414, 53.86180114746094, -17.794679641723633, 19.877429962158203, -12.562736511230469, -64.6925048828125, 2.8934359550476074, -69.30143737792969, 11.538773536682129, 14.924782752990723, -52.80881881713867, -2.074591636657715, -56.12260055541992, 21.129911422729492, -8.312826156616211, -2.5504884719848633, 17.56533432006836, -45.42356491088867, -17.814058303833008, -12.254638671875, 44.600276947021484, 13.581197738647461, 12.439274787902832, 43.27046203613281, -9.573982238769531, -24.167633056640625, 34.11290740966797, 25.806947708129883, -50.10192108154297, -16.083263397216797, -31.825698852539062, 37.30806350708008, -74.23542785644531, -25.844768524169922, -32.725196838378906, 49.50791931152344, -13.957321166992188, 5.0664777755737305, -60.93623733520508, 33.50914764404297, -42.87821960449219, -17.332881927490234, 48.645023345947266, -46.8717041015625, -3.9656991958618164, -1.3447712659835815, 16.733863830566406, 39.4898681640625, 38.34309768676758, 5.752702713012695, -53.1953125, -62.07552719116211, -15.157916069030762, 7.2840094566345215, -1.069746732711792, 6.18395471572876, 65.02326965332031, -16.387386322021484, -2.5423293113708496, -3.673903226852417, 15.262249946594238, -24.712608337402344, 2.2430825233459473, 13.319023132324219, 51.68402099609375, 27.845443725585938, 41.56640625, 0.33503472805023193, 6.595991134643555, 22.21942710876465, -10.534788131713867, 19.833951950073242, -7.155075550079346, -29.355453491210938, 10.356793403625488, 34.50708770751953, 19.067899703979492, 10.189743041992188, -23.361106872558594, 13.783941268920898, 8.229598045349121, 34.99440383911133, 25.604631423950195, 16.409698486328125, -54.08173751831055, 26.82399559020996, 62.68021774291992, -10.500279426574707, -6.200294017791748, -41.28217315673828, 38.328983306884766, 6.292500019073486, 13.978147506713867, 12.418060302734375, 4.060032367706299, -38.58098220825195, 19.01590919494629, -60.644874572753906, -5.91128396987915, -35.86362838745117, 36.754661560058594, 53.41999053955078, 22.669815063476562, 0.34013694524765015, 13.472290992736816, -36.22183609008789, 54.06277847290039, -8.243228912353516, -29.665504455566406, 24.584922790527344, 6.769709587097168, 18.70859146118164, 12.535268783569336, 10.164133071899414, 24.589462280273438, 8.838360786437988, 6.672144889831543, 41.24983596801758, 62.57868957519531, 25.38338851928711, 43.277645111083984, 6.234586715698242, 18.943084716796875, -46.72100067138672, -34.108150482177734, 7.480000972747803, -60.855560302734375, -65.35736083984375, -14.38608169555664, 11.828556060791016, 54.85467529296875, 26.478153228759766, -23.24694061279297, -79.16612243652344, 27.38916015625, -16.381330490112305, -43.9140739440918, 49.358036041259766, 57.431827545166016, 6.110751152038574, 37.875282287597656, 23.6749267578125, -14.809459686279297, -11.83657169342041, -26.088348388671875, -36.55972671508789, -4.047484874725342, 27.834980010986328, 0.879912257194519, 19.713356018066406, -14.614008903503418, -16.527759552001953, 22.942230224609375, 30.472110748291016, -67.02108001708984, -13.85224437713623, 21.4968318939209, 14.481966018676758, -20.61957359313965, -27.343774795532227, 63.71929931640625, -2.7460224628448486, -8.798219680786133, 20.451528549194336, 11.189409255981445, -22.037717819213867, 28.89055824279785, -70.94810485839844, -19.252405166625977, 9.295454978942871, 3.4451918601989746, 46.44846725463867, 7.668334484100342, -9.211906433105469, 2.5203757286071777, 58.09663772583008, 47.97725296020508, -35.80076217651367, -49.20987319946289, -15.392960548400879, 34.099151611328125, -14.574134826660156, -11.32028579711914, -23.602169036865234, -19.7154483795166, 51.29176712036133, 40.63722610473633, 19.201431274414062, -22.823631286621094, 30.379924774169922, 33.554073333740234, 16.891796112060547, -11.394067764282227, -32.57916259765625, -73.66461181640625, 8.24036693572998, 18.149152755737305, -43.653175354003906, -44.3720817565918, 51.157779693603516, 0.7101781368255615, 66.10208129882812, -10.569793701171875, -31.741374969482422, 35.95429992675781, -30.579513549804688, 5.934442043304443, 4.039989471435547, -35.4783935546875, 26.691164016723633, 3.7453060150146484, 33.396854400634766, -35.498287200927734, -9.072662353515625, 13.393646240234375, 30.498096466064453, -25.075590133666992, 1.7479456663131714, -28.03358268737793, 19.20011329650879, 7.175291061401367, -30.684284210205078, 18.498563766479492, -19.391632080078125, 104.32752990722656, -22.390113830566406, 24.582637786865234, -7.206121444702148, -48.45830154418945, 63.382877349853516, 6.338571548461914, 35.12068176269531, 17.307464599609375, 5.623431205749512, 1.1126489639282227, 25.462011337280273, -41.585567474365234, -1.2464348077774048, 20.486757278442383, -31.426626205444336, -25.13483428955078, 29.022356033325195, 6.197873115539551, 71.09729766845703, 29.502798080444336, -36.83687210083008, -23.33562660217285, 33.4754638671875, -40.69132995605469, 15.51030445098877, 4.7485551834106445, -15.428054809570312, 21.875844955444336, 38.68512725830078, 25.911121368408203, -13.850652694702148, 24.50716781616211, 2.733280658721924, 4.73943567276001, -19.37848663330078, -17.79703140258789, -18.837303161621094, -46.670867919921875, 30.195091247558594, 1.8536875247955322, -10.859755516052246, 4.34529972076416, 49.91269302368164, -22.00486183166504, -22.194786071777344, 1.9947123527526855, 46.00685119628906, -5.887533664703369, 3.2670464515686035, 21.892372131347656, -24.170347213745117, -22.26081085205078, 1.9255142211914062, -10.537125587463379, -12.778932571411133, -31.282485961914062, 35.54549789428711, -0.25495538115501404, -12.5722074508667, -35.32853698730469, 32.816627502441406, 27.81696891784668, -3.365262746810913, 55.867774963378906, -46.84730911254883, 3.900087594985962, -84.95557403564453, -7.519268989562988, -18.30059051513672, -6.524116516113281, 50.06884765625, -7.736135482788086, -46.137454986572266, 22.023014068603516, 14.649003982543945, 69.27468872070312, -27.176910400390625, -42.242366790771484, 24.668153762817383, -14.704737663269043, 27.45489501953125, 40.146934509277344, -14.630388259887695, 18.414785385131836, -36.16252136230469, 9.942795753479004, 72.28656005859375, 18.057964324951172, -17.8630313873291, 17.547849655151367, -6.528015613555908, 6.090698719024658, -31.573486328125, -27.887189865112305, -14.536810874938965, 24.34471321105957, -21.3619441986084, 23.774551391601562, 18.474964141845703, 7.160058498382568, 41.138519287109375, 17.366456985473633, 81.95785522460938, -23.829113006591797, 7.562575817108154, 9.833904266357422, -3.7156145572662354, -19.957122802734375, 18.687232971191406, -22.69350814819336, 26.025928497314453, -52.4734001159668, -14.414400100708008, -28.416879653930664, 12.752323150634766, 17.23655891418457, 70.74722290039062, 29.232723236083984, 6.302405834197998, 39.589969635009766, 1.9550613164901733, -2.525313377380371, 45.08446502685547, -22.901329040527344, -70.57209777832031, -27.75273895263672, -3.8413403034210205, -14.523280143737793, 2.4788413047790527, -75.44364166259766, -34.09347152709961, -0.3602064251899719, -45.3754768371582, -15.811676025390625, 2.8463315963745117, 3.4804654121398926, -11.83017635345459, 60.69218063354492, -53.41038513183594, 51.80337905883789, -46.22864532470703, -6.825708389282227, 23.702674865722656, 5.218452453613281, -19.410594940185547, -40.56657791137695, 9.354538917541504, -19.2590389251709, 30.849634170532227, -9.50208854675293, 29.016460418701172, -11.273715019226074, 30.120288848876953, 0.44446858763694763, 28.052705764770508, -33.84799575805664, -14.28757095336914, 10.494003295898438, -15.20675277709961, -13.102953910827637, -31.211931228637695, 24.81674575805664, 20.30097198486328, 37.40736770629883, 4.516452789306641, -14.478042602539062, -48.27496337890625, 3.2463302612304688, -16.37033462524414, 27.47651481628418, 6.687614917755127, -57.17281723022461, -16.629785537719727, -4.488681316375732, -39.23245620727539, 22.316967010498047, -10.257195472717285, -119.95500946044922, 31.48146629333496, 14.685709953308105, -42.37537384033203, 11.839612007141113, 37.77825164794922, -35.725223541259766, 40.7154426574707, -19.041690826416016, -27.37875747680664, 11.728336334228516, 31.235858917236328, 11.018263816833496, -15.988173484802246, 23.077117919921875, -3.458320379257202, 2.825300455093384, -0.9542332887649536, 7.781033515930176, 9.205164909362793, 48.40120315551758, -36.98106384277344, -13.723245620727539, -29.0888614654541, -39.01201248168945, -14.992918014526367, -20.184627532958984, -27.26799964904785, 40.17470932006836, 14.518997192382812, 23.64703369140625, -4.502290725708008, -21.555370330810547, -23.179899215698242, -6.225390911102295, -7.366725921630859, 31.265451431274414, -41.18873977661133, -37.35757064819336, -20.030746459960938, 15.594808578491211, 29.393522262573242, 18.233366012573242, -44.31676483154297, -16.726604461669922, -9.35219669342041, 16.673076629638672, -7.267784595489502, 38.19722366333008, 9.608052253723145, -45.032470703125, -14.735061645507812, 37.905601501464844, -34.346256256103516, 12.182063102722168, 50.092586517333984, 17.936738967895508, -54.767333984375, 4.2312397956848145, 13.175623893737793, 16.5556640625, -13.827143669128418, -21.192792892456055, 25.363754272460938, 7.175971031188965, 38.2457160949707, 19.06317901611328, -15.710359573364258, 9.595335006713867, 28.105941772460938, -23.377721786499023, -22.743850708007812, 2.038646697998047, 46.97795486450195, 6.530550479888916, -15.934721946716309, 2.521350622177124, 43.995479583740234, -36.35240936279297, 3.0091118812561035, -19.85531234741211, 42.26205825805664, -19.988122940063477, 10.385761260986328, -20.183902740478516, -42.228328704833984, -1.715388536453247, -2.916675329208374, 34.83932876586914, -38.84370803833008, -26.264034271240234, 2.171412467956543, 25.846696853637695, 50.30519485473633, 43.22509765625, 16.996021270751953, 7.268805027008057, 17.483768463134766, -28.130863189697266, -13.600746154785156, -39.04743957519531, 15.744636535644531, -12.594927787780762, -23.764568328857422, -43.828392028808594, 26.170228958129883, 15.776418685913086, -5.791042804718018, -20.5470027923584, 10.923178672790527, -20.532503128051758, 8.508038520812988, 33.36064147949219, 41.449180603027344, 34.92067337036133, -4.186671733856201, 10.608494758605957, 45.79814529418945, 7.740215301513672, -2.960911750793457, -58.587032318115234, -12.002463340759277, 53.759605407714844, -39.892513275146484, 58.734764099121094, -7.396451950073242, 8.966282844543457, 5.632663249969482, -7.449042320251465, 6.3251261711120605, 6.3793559074401855, 42.929664611816406, -69.05525970458984, 29.8007755279541, -35.60783004760742, -23.638826370239258, -1.1932830810546875, 10.070818901062012, -28.34318733215332, 50.02023696899414, -33.919151306152344, -44.08153533935547, -17.605587005615234, -50.62596130371094, -2.864638328552246, -25.76007080078125, 29.014137268066406, 6.493536949157715, 1.6635279655456543, -44.55752944946289, -6.531939506530762, -28.101314544677734, -27.71234130859375, 29.42390251159668, 25.74117088317871, 1.7277084589004517, -8.996955871582031, -11.295242309570312, -19.643741607666016, 5.689535140991211, -2.91727876663208, -0.13941869139671326, 21.691455841064453, -37.266578674316406, -2.5046277046203613, -16.772974014282227, -9.490731239318848, 30.973541259765625, -17.755245208740234, -22.205581665039062, 8.452917098999023, 18.528715133666992, 55.97537612915039, 28.652462005615234, -8.17822265625, 21.49818229675293, -25.408241271972656, -29.67610740661621, 29.260154724121094, -14.814048767089844, 18.754261016845703, 9.077348709106445, -23.335634231567383, 6.7882981300354, 27.001699447631836, -14.800271987915039, 49.174400329589844, 23.045352935791016, 39.88640213012695, 55.251895904541016, 12.142735481262207, -43.86732482910156, 12.908642768859863, 37.59624099731445, 7.623184680938721, -68.59120178222656, 11.125070571899414, -27.795095443725586, -28.487716674804688, -34.84710693359375, -26.071138381958008, 50.47008514404297, 51.56087112426758, 5.533950328826904, 14.87848949432373, 0.07100431621074677, -48.959232330322266, 32.03585433959961, -11.969330787658691, 69.85344696044922, 19.28017807006836, 21.321060180664062, 35.951332092285156, -27.99522590637207, 3.6679060459136963, 27.63111114501953, 46.512142181396484, -28.1697940826416, -35.36711120605469, 50.57268524169922, -11.82383918762207, 3.063721179962158, 23.06998062133789, -4.333272933959961, -19.252578735351562, -21.53567886352539, -15.659635543823242, 29.90035629272461, -29.690025329589844, 35.12664031982422, 1.1300263404846191, -7.654752254486084, -17.949682235717773, 10.328946113586426, -15.234138488769531, 3.221667766571045, -19.679012298583984, -47.30361557006836, 28.191186904907227, 8.088743209838867, 23.25596046447754, 6.680333614349365, -7.978242874145508, -18.22502326965332, -14.454339027404785, 9.951665878295898, -13.620903968811035, 46.087093353271484, 20.00489616394043, -15.591745376586914, -26.38590431213379, 9.54610824584961, -100.50919342041016, 16.871498107910156, 45.3232307434082, 24.92032241821289, -2.644454002380371, 37.322898864746094, 20.507423400878906, -11.337981224060059, -15.667779922485352, -22.723970413208008, 2.862170696258545, 32.08961868286133, -24.659648895263672, 76.76337432861328, -27.782554626464844, -6.84285831451416, -20.081769943237305, -14.465049743652344, 31.065719604492188, -28.0595645904541, 15.749979019165039, -16.809118270874023, 27.793249130249023, 4.788531303405762, 29.02366828918457, -28.985244750976562, 1.498812198638916, 41.111839294433594, 17.71611785888672, 22.665538787841797, 19.331443786621094, -14.114169120788574, 28.552093505859375, 14.75601863861084, -38.16936492919922, 54.71135330200195, -7.202035903930664, 39.991241455078125, 27.31517791748047, 10.291934967041016, -3.978070020675659, 50.592987060546875, 24.15540885925293, 13.22856616973877, -12.855958938598633, 2.3398537635803223, -23.28787612915039, -14.967910766601562, -44.48319625854492, -29.829408645629883, 37.00753402709961, 22.247331619262695, -75.29953002929688, 5.645231246948242, 55.96553421020508, 76.85746002197266, 12.325531959533691, -39.757259368896484, 42.01115036010742, -56.41636657714844, 29.065744400024414, -20.730857849121094, -5.671128749847412, -31.184846878051758, 39.93020248413086, 1.371233582496643, -27.623695373535156, -13.369555473327637, 63.2111930847168, 3.3768997192382812, -43.58905792236328, 8.340871810913086, -17.923133850097656, 43.62101745605469, -85.43597412109375, 6.459315776824951, 26.811050415039062, -1.0676976442337036, -59.64335250854492, -25.90192985534668, 37.666221618652344, -39.601863861083984, -51.03670120239258, -43.26787185668945, -64.23798370361328, -11.373329162597656, 2.872332811355591, 10.746447563171387, -35.746437072753906, -31.681577682495117, 20.52953338623047, -11.497608184814453, -8.428689956665039, 34.8764533996582, -30.53194808959961, -39.30204391479492, 1.8646830320358276, 0.40680238604545593, 3.271146059036255, 23.70791244506836, -24.44548225402832, 17.20269775390625, -41.58736038208008, 28.744836807250977, 35.8636589050293, -3.2683420181274414, 0.9076597094535828, -35.7531852722168, 1.1705904006958008, 6.1105637550354, 6.719335079193115, 31.926130294799805, -17.453168869018555, -29.83438491821289, 20.8420467376709, 29.4694881439209, -12.039569854736328, 23.984535217285156, 12.486769676208496, -6.194611549377441, -33.766448974609375, 57.76762771606445, 7.2682085037231445, 19.298076629638672, -23.848388671875, -30.563583374023438, 39.258155822753906, 11.07474136352539, 21.802675247192383, 7.397871971130371, -5.630937099456787, -29.609867095947266, -7.8277668952941895, -0.1970839500427246, 33.801300048828125, 41.20520782470703, -23.126319885253906, 26.595468521118164, 9.499963760375977, -2.476392984390259, 51.77345275878906, -9.181774139404297, -3.0858583450317383, 24.03463363647461, -4.76065731048584, 53.02479553222656, -15.636941909790039, 0.005717366933822632, -21.743322372436523, 2.1655383110046387, 30.723323822021484, -15.760444641113281, 27.210111618041992, 0.4633665978908539, 7.074354648590088, -24.114099502563477, 20.262746810913086, -20.672136306762695, -32.554996490478516, -53.0782470703125, 57.551822662353516, -0.8153494596481323, 7.7116217613220215, 41.282379150390625, -30.874242782592773, 84.87875366210938, -23.59928321838379, -19.77933120727539, -35.01382827758789, 12.897241592407227, 20.118074417114258, 41.940673828125, 7.729005813598633, -58.52118682861328, -4.400897026062012, 58.77327346801758, -33.115352630615234, 18.515403747558594, 5.510829448699951, 37.424381256103516, -8.609121322631836, 8.589848518371582, 10.736522674560547, -18.967397689819336, -17.592803955078125, -12.931570053100586, -39.65180969238281, -16.959209442138672, 31.605493545532227, -29.50040054321289, 16.059568405151367, 38.30838394165039, 18.255901336669922, -3.1294498443603516, 55.320064544677734, 6.944732666015625, 8.785787582397461, -51.496891021728516, -27.035594940185547, -1.0962727069854736, 27.95414161682129, 2.271395444869995, 13.425030708312988, 17.072986602783203, -10.403386116027832, 29.802085876464844, -17.487165451049805, 14.321072578430176, -5.292422294616699, -54.37604522705078, -4.776828765869141, 36.76628494262695, 29.474149703979492, 15.277299880981445, -1.3892536163330078, -10.27895736694336, -3.295215606689453, -70.78273010253906, -17.058584213256836, 11.594537734985352, -6.3838324546813965, -31.20579719543457, 53.34455490112305, 8.471982955932617, -12.252914428710938, -38.950225830078125, 5.884946346282959, 5.682897090911865, -82.78741455078125, 14.572364807128906, 17.637508392333984, 31.03946304321289, -60.69624328613281, 25.19753646850586, 2.5633699893951416, 10.420817375183105, -46.34799575805664, -6.5591254234313965, 33.03577423095703, 4.404843807220459, 10.629344940185547, -2.0813865661621094, -9.878066062927246, 62.61491775512695, 1.2237558364868164, 9.616698265075684, 18.949480056762695, 46.958805084228516, 9.073644638061523, 11.965104103088379, -26.10322380065918, 33.9034423828125, 46.7923583984375, 2.5307602882385254, -15.699695587158203, 45.77644348144531, 66.2762222290039, -12.075483322143555, -16.200767517089844, -3.4330902099609375, 41.23843002319336, -8.55005931854248, -41.92475128173828, -16.7711124420166, 8.988847732543945, 22.289287567138672, -34.22453689575195, -10.10856819152832, -23.433704376220703, 17.968460083007812, -37.67227554321289, 48.50078201293945, -7.708010673522949, -51.30672073364258, 26.333345413208008, -50.35049819946289, 18.056150436401367, -22.25560188293457, -4.087263584136963, -44.64380645751953, 0.26136624813079834, 20.959278106689453, 2.2747883796691895, -14.732406616210938, -37.603424072265625, -11.79990005493164, 21.962411880493164, 35.105628967285156, -11.495599746704102, -59.753211975097656, -2.2688121795654297, -71.22248840332031, 38.8927116394043, 26.890073776245117, -46.54702377319336, -15.60207462310791, -9.269979476928711, -37.52684783935547, -6.329872131347656, -9.806859970092773, 44.1513786315918, 13.36806583404541, -2.3480300903320312, 35.012367248535156, -7.931840896606445, 68.0941162109375, 23.26299476623535, -2.6495063304901123, -1.5460890531539917, 17.403247833251953, 6.538773536682129, -35.010807037353516, -54.73530578613281, -10.403271675109863, -28.287885665893555, 39.27589416503906, -22.355627059936523, 36.15171813964844, -71.44947814941406, -4.4216837882995605, 52.780738830566406, -10.079408645629883, 33.06425857543945, -23.6197566986084, 22.863061904907227, 23.310117721557617, -10.439197540283203, -70.15050506591797, 3.0188326835632324, 73.54132843017578, -16.593299865722656, 56.37614059448242, -20.235179901123047, 37.4320068359375, 10.360847473144531, -20.255126953125, -17.234651565551758, -5.834831237792969, -30.066059112548828, 36.52017593383789, -4.930078983306885, -9.002135276794434, -11.257684707641602, -2.595360517501831, -2.7213234901428223, -23.618640899658203, 9.16958236694336, -13.312921524047852, -46.859840393066406, -0.835456371307373, 42.79880905151367, -52.408023834228516, 13.217910766601562, 35.89725112915039, 72.36416625976562, 9.367698669433594, 13.239431381225586, 46.2713623046875, -8.085641860961914, -3.091383934020996, 1.1234476566314697, -54.11564636230469, -1.634680986404419, -9.622654914855957, 8.493355751037598, 12.637530326843262, -3.1312856674194336, -41.40946960449219, -9.073177337646484, -24.689481735229492, -45.736717224121094, 42.811614990234375, 10.50922679901123, 3.9211232662200928, -44.60719299316406, -48.91339111328125, 25.65915298461914, -23.10137939453125, 17.21088409423828, -7.064123630523682, -7.937429904937744, -29.319522857666016, 15.178112030029297, 5.249109745025635, -39.55009460449219, -23.60361099243164, -1.1808149814605713, 18.899999618530273, 5.6267313957214355, -78.02410888671875, -11.944294929504395, -33.87877655029297, -34.214073181152344, -14.981439590454102, 0.7510221004486084, -48.8663330078125, 46.1669807434082, -17.84442901611328, 23.788585662841797, 15.055618286132812, -32.44598388671875, 2.444277763366699, -13.539963722229004, -15.10659122467041, 44.51561737060547, -18.67647933959961, -4.37108850479126, -13.032841682434082, -31.22154998779297, 10.930647850036621, -6.316664218902588, -32.05034637451172, 30.700685501098633, -0.23922324180603027, -48.26865768432617, 30.345428466796875, 45.325138092041016, -68.74575805664062, -1.347187876701355, -19.280519485473633, -31.161243438720703, -17.766868591308594, 26.207412719726562, 32.38983154296875, -3.4369702339172363, 19.655427932739258, 43.91609191894531, -15.807310104370117, 27.056589126586914, -23.26401138305664, 22.05311393737793, 49.17839813232422, -14.718997955322266, 6.5536651611328125, -9.097929000854492, 32.36708068847656, -30.81791877746582, -25.251359939575195, -44.96118927001953, 25.613718032836914, 52.717288970947266, -23.167930603027344, 18.6190185546875, 9.525636672973633, -27.21562957763672, -4.134554862976074, 9.343156814575195, -23.218175888061523, -17.729156494140625, 20.703067779541016, -32.706077575683594, 17.00266456604004, 8.2639799118042, -38.03643798828125, -0.8632074594497681, 36.25172805786133, -11.638131141662598, -44.43292236328125, -51.63139343261719, 2.918968439102173, -8.1676607131958, -18.20752716064453, 5.277615547180176, 7.94760799407959, 23.878808975219727, -18.457012176513672, 34.985599517822266, -30.758167266845703, -2.871542453765869, 14.726414680480957, -6.575211524963379, 9.029996871948242, -21.746065139770508, 6.023571014404297, 46.52180480957031, 23.795806884765625, -42.98701477050781, -26.501432418823242, 6.295118808746338, -14.25400447845459, -13.311683654785156, 4.9319233894348145, -22.23757553100586, -27.333343505859375, -49.75505065917969, -59.321067810058594, 32.0324821472168, 16.566661834716797, 40.10305404663086, 28.36849021911621, -57.08121109008789, 41.31132507324219, 14.799463272094727, -37.87545394897461, 9.434983253479004, -30.523935317993164, 25.588428497314453, -50.998233795166016, -43.403114318847656, 3.4848616123199463, -49.8179931640625, -48.31141662597656, 3.57914400100708, 16.820919036865234, -31.739805221557617, 40.5416145324707, 43.81355667114258, -5.956656455993652, 10.863821029663086, -11.973250389099121, -11.598197937011719, 12.840253829956055, -13.87727165222168, -14.144526481628418, -10.662747383117676, -10.564987182617188, -29.381010055541992, -2.492410182952881, -22.979297637939453, -23.85099983215332, -14.1887845993042, 42.947601318359375, 19.04271125793457, 9.324822425842285, -4.265084266662598, 0.6150574088096619, -8.672605514526367, -15.740805625915527, 27.252304077148438, 26.297117233276367, -29.676738739013672, -39.320247650146484, 41.2954216003418, -21.427656173706055, -35.15660095214844, 8.166919708251953, 33.356170654296875, 36.9847297668457, -4.744365215301514, 22.82430648803711, -27.72161102294922, -22.992115020751953, -34.58636474609375, 22.670207977294922, -5.4074296951293945, -25.231569290161133, 4.863247871398926, 24.511375427246094, -96.23530578613281, 18.180355072021484, -12.503837585449219, 39.20628356933594, -12.392827987670898, 18.79555892944336, 10.032208442687988, 14.207582473754883, -22.604965209960938, 44.89232635498047, 0.5467942357063293, -29.623756408691406, 1.5842245817184448, 52.678131103515625, -0.9708380699157715, 16.697113037109375, -30.71742820739746, -26.73365592956543, -21.728763580322266, -5.805679798126221, -9.850431442260742, 13.442934036254883, 33.05793762207031, -18.121631622314453, 18.457504272460938, -29.69363784790039, 2.4850525856018066, -0.5915796756744385, 11.222818374633789, -44.16360855102539, 18.172555923461914, 45.186256408691406, -43.4085807800293, 65.82988739013672, -28.968355178833008, 20.83900260925293, 17.632535934448242, -25.960193634033203, -29.867355346679688, 52.24544143676758, 24.78900146484375, 26.165958404541016, -14.634430885314941, 51.481021881103516, -10.684582710266113, -7.90594482421875, 36.95882797241211, -0.12636446952819824, 32.93952560424805, 2.0193026065826416, 19.59360694885254, 33.66799545288086, -8.903411865234375, 17.324655532836914, 37.68550109863281, -23.303447723388672, 36.54888153076172, 33.10243225097656, 47.80449676513672, 10.446259498596191, -1.929136872291565, -12.622861862182617, 10.585418701171875, -26.821075439453125, 0.13623476028442383, 16.10186004638672, 65.95758819580078, 5.165897846221924, -14.676933288574219, 40.7056999206543, -11.641300201416016, -21.694246292114258, -12.068033218383789, -38.63568115234375, -2.9422240257263184, 21.312217712402344, -32.58572006225586, -57.555946350097656, 10.001770973205566, -28.442359924316406, 24.791545867919922, -11.186480522155762, 78.78653717041016, 29.459701538085938, 14.51640510559082, 25.75904655456543, 53.329490661621094, -49.77238082885742, 19.063838958740234, -18.77784538269043, 20.740678787231445, -11.440411567687988, -10.65790843963623, -2.299783229827881, 20.259227752685547, 24.367910385131836, 33.31696319580078, 39.70792007446289, 8.632220268249512, -50.097415924072266, -41.418190002441406, -2.861619472503662, -79.38819122314453, -39.78300476074219, -42.04789733886719, -7.045474052429199, -4.655486106872559, -61.13666534423828, 39.9907112121582, 16.25497817993164, 17.735605239868164, -32.669715881347656, 10.302277565002441, -31.18194580078125, 6.317259788513184, 9.432147026062012, 6.193647384643555, 21.663387298583984, 24.570594787597656, -37.08639907836914, 3.2405145168304443, -0.06438794732093811, 12.462248802185059, -9.97787857055664, 37.98625564575195, -1.3088363409042358, -2.2899107933044434, 12.493066787719727, -10.037287712097168, -5.602104187011719, 14.083142280578613, -33.773563385009766, 43.639469146728516, 21.450660705566406, -8.258952140808105, -70.53217315673828, 20.758663177490234, -33.1223030090332, 15.663771629333496, 54.573246002197266, 78.74327087402344, 26.05617904663086, 44.016475677490234, 0.1992768943309784, -46.634918212890625, 21.214981079101562, -6.023209095001221, 10.76916790008545, 18.83489418029785, -20.798704147338867, -0.06897836923599243, -4.9082746505737305, 25.259578704833984, -42.94089889526367, 26.490825653076172, 55.66289520263672, -42.898643493652344, -6.954975605010986, -20.898252487182617, -9.140888214111328, -22.51474952697754, 3.0759103298187256, -10.224637985229492, 22.345712661743164, -40.302120208740234, -41.91325378417969, 22.636247634887695, -6.414801597595215, -19.839956283569336, -54.249446868896484, 2.4397330284118652, 9.234010696411133, -71.09388732910156, -37.10154724121094, -8.566655158996582, -20.77086067199707, -37.09717559814453, 1.62149977684021, -21.87178611755371, -7.86384391784668, -54.697471618652344, -27.575963973999023, 18.593847274780273, -45.740482330322266, 26.466779708862305, 16.241466522216797, 53.35328674316406, -32.27006912231445, -1.3865993022918701, 44.64979553222656, -0.39399605989456177, 17.37752342224121, -2.678429126739502, -2.1077117919921875, 18.93720245361328, -27.09670639038086, 48.8244514465332, 17.38139533996582, 25.272083282470703, -72.96935272216797, 24.575063705444336, 1.6802172660827637, 27.75840187072754, -4.514134407043457, -22.572500228881836, -13.371257781982422, 32.74364471435547, -38.716156005859375, 4.524871826171875, -16.201171875, -16.595335006713867, 9.735549926757812, 8.626219749450684, 18.690643310546875, 22.964757919311523, -17.218908309936523, -19.12571907043457, -38.43891906738281, -58.50536346435547, 25.67500114440918, -18.724586486816406, -44.998291015625, -41.169921875, -7.395153045654297, 17.3985595703125, -35.1690559387207, -30.169471740722656, 52.840877532958984, -0.1299191415309906, -51.085636138916016, 31.534116744995117, -4.160079002380371, 11.468186378479004, 41.33423614501953, -16.20473861694336, -0.7977113723754883, -21.50786018371582, 27.978368759155273, -1.987089991569519, -47.9134521484375, -16.009553909301758, 11.965533256530762, -3.7898318767547607, -8.216288566589355, 29.525188446044922, -7.180303573608398, 9.956741333007812, -42.991268157958984, -51.924015045166016, -28.606285095214844, -26.607189178466797, -43.68411636352539, -7.509084224700928, 50.737548828125, 22.927873611450195, -2.3839473724365234, -13.189323425292969, -56.08647537231445, 1.2982866764068604, 14.071267127990723, -28.268465042114258, -4.289719104766846, -37.56399917602539, -19.537954330444336, 12.1441011428833, 16.048709869384766, 12.125518798828125, -3.8068933486938477, 6.258930683135986, 29.80840301513672, 3.8863067626953125, 28.297115325927734, -16.753080368041992, -2.004135847091675, -10.668370246887207, 38.239131927490234, 29.6978702545166, 17.954469680786133, -22.144336700439453, -13.544588088989258, 25.861610412597656, 11.396597862243652, 27.194787979125977, 29.469707489013672, 57.0370979309082, -0.9945436716079712, 9.196613311767578, 5.583183288574219, -14.764039039611816, -16.33664321899414, -21.428325653076172, 1.8487632274627686, 27.240690231323242, 44.854339599609375, 74.98485565185547, -15.327635765075684, -18.877973556518555, 56.714542388916016, -28.719083786010742, -14.781545639038086, 46.646522521972656, 25.747821807861328, 3.336954116821289, -4.414910793304443, 27.667287826538086, 4.152466773986816, -71.63998413085938, 18.1796817779541, 12.498774528503418, 35.982940673828125, -15.154154777526855, -23.2589054107666, -4.39402961730957, 37.24140167236328, -45.42236328125, 0.2116008996963501, 17.713863372802734, 5.531586647033691, -21.039365768432617, 5.195487022399902, 1.0080010890960693, -6.119129180908203, 36.04378890991211, -15.396500587463379, -17.367521286010742, -1.1678519248962402, 36.96018981933594, -4.056823253631592, 28.90580177307129, 23.360490798950195, -36.7901725769043, 0.4009645879268646, 3.822995901107788, 6.464316368103027, -13.488420486450195, 42.00864791870117, 20.63884162902832, -37.05319595336914, -27.130006790161133, 10.930604934692383, -26.593921661376953, 7.453842639923096, 55.59891128540039, -28.74469757080078, 28.948041915893555, -18.747020721435547, -22.803937911987305, -8.18510913848877, -11.269600868225098, -3.475106954574585, -20.04972267150879, 16.19339942932129, -26.367618560791016, -11.923434257507324, 12.830650329589844, 60.014892578125, 6.872555255889893, 31.05389404296875, -1.4984064102172852, -29.02442741394043, -12.996084213256836, -56.56956100463867, 8.568004608154297, -0.9867980480194092, 16.572811126708984, 2.212344169616699, -2.8549892902374268, -59.787593841552734, 28.817291259765625, 11.126626968383789, -44.337867736816406, 15.092639923095703, 12.708711624145508, 26.53714942932129, -54.76300811767578, -1.5970884561538696, 43.8297233581543, 15.38750171661377, -28.174848556518555, 14.62566089630127, 1.981584906578064, -28.145015716552734, -52.472434997558594, -24.24896812438965, -31.062265396118164, 14.506876945495605, 16.544296264648438, 4.774775981903076, -15.79715347290039, 35.40010070800781, -56.00201416015625, -23.65808868408203, 12.554890632629395, -26.155168533325195, 47.51895523071289, -57.8652458190918, -24.454927444458008, -22.178157806396484, 29.315349578857422, 36.13311767578125, -60.75326156616211, -36.119110107421875, 14.441934585571289, 27.497819900512695, -13.691106796264648, 7.342889785766602, 1.6608004570007324, 17.78116798400879, -7.832699298858643, -38.680179595947266, 55.85261917114258, 1.652707815170288, -66.10614013671875, 39.44700241088867, -4.45379114151001, 43.04826354980469, -16.994781494140625, -5.964705467224121, 35.80024337768555, -7.490462303161621, -36.80258560180664, -12.58367919921875, -26.214935302734375, 19.452957153320312, -37.206565856933594, 46.42632293701172, 16.05864143371582, 1.7475833892822266, -1.1301435232162476, -0.7301228046417236, 10.165553092956543, -45.1553955078125, -16.503559112548828, -38.85144805908203, -31.310853958129883, 23.148422241210938, -43.71074676513672, -26.21430206298828, -42.30527877807617, -52.8704948425293, -16.415756225585938, 21.069459915161133, -24.534849166870117, 4.856931686401367, -27.683984756469727, 19.03762435913086, 11.754032135009766, -12.077152252197266, 8.052730560302734, -7.496418476104736, 32.30084991455078, -23.607994079589844, 1.749719262123108, -3.6381630897521973, 11.968482971191406, -10.59526252746582, 29.40821647644043, 22.367862701416016, -30.55315399169922, -7.945444107055664, -9.624545097351074, -13.24912166595459, -36.56172180175781, 14.235113143920898, -26.84241485595703, -17.01947784423828, 22.052743911743164, 48.128875732421875, -0.3232530951499939, -3.302816152572632, -46.99142837524414, -2.1203248500823975, 30.384502410888672, -21.330265045166016, 40.7306022644043, -2.9221911430358887, -29.738466262817383, -26.17901039123535, 32.94063186645508, 9.359987258911133, -5.485519886016846, 3.4201393127441406, -6.991150856018066, 10.545524597167969, -19.650283813476562, 0.21554657816886902, 13.825411796569824, -41.197208404541016, 14.72647476196289, 26.003433227539062, 24.60456085205078, 6.947986125946045, -60.168983459472656, -35.844451904296875, 26.52255630493164, 53.605384826660156, 27.83513641357422, -32.377471923828125, 13.650199890136719, -41.8714599609375, 5.612704753875732, 21.384220123291016, 52.16571044921875, 33.701560974121094, -14.123969078063965, -49.60652542114258, -0.038211967796087265, -13.984891891479492, -2.1242568492889404, 49.249961853027344, 39.228607177734375, 10.130248069763184, -15.57607650756836, -29.080350875854492, -6.390125751495361, -61.50905990600586, -39.76665496826172, 26.134517669677734, 90.83824920654297, 26.703027725219727, 22.345006942749023, 26.79640007019043, -7.430731296539307, 8.153326034545898, -12.290099143981934, 2.0334181785583496, -9.420389175415039, -66.89170837402344, 44.82637023925781, -17.459503173828125, 6.539038181304932, 45.998531341552734, -70.07878875732422, -1.7651245594024658, 29.016286849975586, 1.2432743310928345, 14.422147750854492, 32.32149124145508, 3.8945891857147217, -27.849525451660156, -29.515451431274414, -0.09388267993927002, -12.610514640808105, 31.37912940979004, 22.253782272338867, 12.378192901611328, 14.119922637939453, -5.537147045135498, 9.287577629089355, 35.228729248046875, -19.191082000732422, 27.359718322753906, -9.826542854309082, -27.776159286499023, 15.549962997436523, -5.315698146820068, 35.030357360839844, -8.275038719177246, -55.10575485229492, 35.445533752441406, -4.406700611114502, -3.7363839149475098, 50.9083366394043, 61.059539794921875, -0.6868956685066223, -18.853878021240234, -14.470620155334473, 6.227747917175293, 8.124505996704102, 48.94366455078125, 16.414989471435547, -6.254947185516357, 10.442704200744629, -11.371809005737305, 41.247013092041016, -8.672342300415039, 74.66941833496094, 17.459850311279297, 42.79314041137695, 9.117020606994629, -20.01327896118164, -39.138427734375, -8.42859172821045, -13.578387260437012, 34.38584899902344, -58.3798942565918, -18.752281188964844, 18.5948543548584, 24.62514877319336, -16.418560028076172, -12.00898265838623, -24.245668411254883, -12.065597534179688, -4.946826934814453, 21.80984115600586, 33.992855072021484, -8.717816352844238, 24.25741195678711, 22.96124839782715, 20.270782470703125, -51.76286315917969, -28.91102409362793, 32.71527862548828, -69.42081451416016, -50.176753997802734, -32.339717864990234, -51.61106491088867, 28.2639102935791, -3.397394895553589, 11.281705856323242, 21.279386520385742, 8.553033828735352, -18.77062225341797, 2.520458459854126, -0.6592434048652649, 4.942477226257324, -30.4966983795166, 43.58125305175781, -21.41712760925293, 8.156676292419434, -56.78271484375, 20.12359619140625, -23.554399490356445, -7.865456581115723, -16.445640563964844, -15.931692123413086, -29.605798721313477, 3.4921493530273438, -45.77254104614258, -46.203208923339844, 13.268139839172363, -4.384059906005859, -11.445630073547363, -20.01166534423828, 8.410821914672852, 7.231269359588623, -10.412670135498047, -18.1463565826416, 16.633808135986328, -54.03944778442383, 0.9757641553878784, -11.36862564086914, 19.820932388305664, -9.226245880126953, -73.021728515625, -26.718046188354492, -17.07260513305664, 38.17918014526367, -59.32849884033203, -17.61574935913086, 3.639497756958008, -17.49412727355957, 17.476381301879883, 14.707432746887207, 13.411030769348145, 23.66417694091797, 24.672164916992188, -25.544647216796875, -8.149138450622559, 38.505985260009766, -28.01822280883789, 24.09445571899414, 15.844490051269531, -25.49681854248047, 57.832881927490234, 1.8563222885131836, 39.68450164794922, -29.67790985107422, 66.4141616821289, -4.802618503570557, 2.9696383476257324, -66.03822326660156, 0.9390872120857239, -30.827152252197266, 0.47320792078971863, -26.566280364990234, 7.06419038772583, -43.194583892822266, 6.585440635681152, 31.12013816833496, -7.347277641296387, 54.11508560180664, -51.506370544433594, 20.32872772216797, 21.849079132080078, -62.46808624267578, 9.499375343322754, -32.17573928833008, 38.855224609375, -78.13200378417969, 23.679157257080078, -21.298992156982422, 4.969895839691162, 7.668028354644775, 27.47523307800293, 48.46204376220703, 55.037574768066406, 37.724220275878906, -46.51614761352539, 56.817569732666016, -21.47018051147461, -52.572235107421875, 25.36009979248047, -17.36993980407715, -6.2656965255737305, 20.401779174804688, -35.0643310546875, 41.48020935058594, 43.6808967590332, -55.330909729003906, 41.75385665893555, 11.04349136352539, -3.042680263519287, 7.823185920715332, 32.84430694580078, 17.4986515045166, -23.799945831298828, -9.027349472045898, 24.981407165527344, 17.948257446289062, -36.02865219116211, 11.895615577697754, -50.7312126159668, 39.712276458740234, 34.85244369506836, 17.66817283630371, 0.7841933369636536, -17.761775970458984, 6.835463523864746, -9.414475440979004, 55.250614166259766, -10.6619234085083, 42.30332565307617, 68.62915802001953, -20.808717727661133, 34.28053665161133, -54.43279266357422, 2.658918619155884, -34.6684455871582, 13.56188678741455, -75.98387908935547, -5.274643421173096, 63.20048904418945, -0.48778849840164185, 49.49578094482422, 9.557291030883789, 0.16382944583892822, -21.32515525817871, 50.52052307128906, 8.803717613220215, -18.308868408203125, 14.828638076782227, 7.028707981109619, 21.367145538330078, 9.041125297546387, -14.902302742004395, -1.3244887590408325, -13.307775497436523, -10.441899299621582, 80.13706970214844, -25.118408203125, -6.255015850067139, 33.36255645751953, -52.977630615234375, 2.2881979942321777, 44.48574447631836, 12.314444541931152, -24.899309158325195, 7.828855991363525, 23.280357360839844, 38.93614196777344, 10.241203308105469, -0.826418399810791, 35.5545539855957, 20.189517974853516, 1.0416008234024048, 23.35308265686035, 25.270631790161133, -6.096380710601807, -29.210186004638672, -33.66828155517578, 0.8041216731071472, -3.4629364013671875, -27.048988342285156, 48.57843780517578, -5.6392292976379395, 10.480985641479492, 7.208422660827637, 4.1432366371154785, 17.338958740234375, 12.114404678344727, 32.683837890625, 1.8045564889907837, -1.4430856704711914, -42.63816452026367, -34.50950241088867, 23.848031997680664, -19.418420791625977, 25.596086502075195, 26.873014450073242, -9.609000205993652, -18.086410522460938, -73.70189666748047, -19.434532165527344, 33.91239929199219, -19.492324829101562, 51.2193489074707, 11.397366523742676, 21.225252151489258, -46.62514877319336, 6.40702486038208, 77.3549575805664, 17.389556884765625, -11.01736068725586, -20.72104263305664, -28.629650115966797, 17.932165145874023, 5.475490570068359, -47.05074691772461, -14.975467681884766, 14.283163070678711, -73.20455932617188, -7.3062591552734375, 54.5185432434082, -21.423288345336914, 13.412535667419434, 32.216224670410156, 17.275005340576172, -1.0230848789215088, 13.429259300231934, 38.853782653808594, 3.371882438659668, -15.890458106994629, 61.764991760253906, -13.738085746765137, -20.41509437561035, 5.1328206062316895, -22.210886001586914, -1.7884548902511597, -0.6647027134895325, -61.32373046875, 13.712387084960938, -16.806833267211914, -6.779487609863281, -19.01654815673828, 6.277106285095215, -22.261137008666992, -31.300189971923828, -18.746349334716797, -7.698960781097412, 5.6267547607421875, -52.27800750732422, 7.202812194824219, 0.1583869457244873, 26.717756271362305, 9.778926849365234, 21.930797576904297, 20.641971588134766, -19.659875869750977, 13.7052640914917, -15.508976936340332, 10.626787185668945, -39.46769332885742, -29.941509246826172, -17.235641479492188, 11.889871597290039, 73.73283386230469, -25.309165954589844, 33.569889068603516, 70.07583618164062, 35.27374267578125, -10.156665802001953, -6.009161472320557, 12.85227108001709, 80.61209869384766, 16.225414276123047, -5.230418682098389, 11.158011436462402, -46.73571014404297, 36.54022979736328, 28.936038970947266, -35.60343933105469, -10.742765426635742, 24.5039005279541, -24.337310791015625, 9.75991153717041, -1.2322334051132202, 34.88653564453125, 16.817575454711914, 0.6707039475440979, -15.440305709838867, 18.894638061523438, -18.108016967773438, 1.4228785037994385, -5.770381450653076, 22.470985412597656, -7.5546979904174805, -45.99864959716797, 1.9438891410827637, -32.261924743652344, 5.131648063659668, -23.653932571411133, 28.605257034301758, -31.853670120239258, 21.611995697021484, 75.56844329833984, 12.445211410522461, 14.611894607543945, -34.08915710449219, -17.305091857910156, 29.179988861083984, -11.224883079528809, 19.77499008178711, -12.177924156188965, 3.8871188163757324, -38.57858657836914, -58.494224548339844, -7.584117412567139, -44.71465301513672, 16.11014175415039, -1.7397485971450806, -10.238382339477539, -24.63797378540039, 25.7885799407959, -9.759868621826172, -39.163246154785156, -34.20358657836914, -4.386224746704102, 42.029014587402344, 6.001343250274658, 34.640830993652344, 12.46883487701416, -14.244590759277344, 7.245624542236328, -11.774332046508789, -37.61422348022461, -11.03673267364502, 2.1123099327087402, -55.984867095947266, 4.908627510070801, 8.64028263092041, -9.498849868774414, -12.807985305786133, -37.44094467163086, 18.385866165161133, -60.42323303222656, -26.49580955505371, 10.033397674560547, 15.804814338684082, -55.646366119384766, 12.15876293182373, 12.881197929382324, 72.944091796875, 50.246009826660156, -19.274507522583008, -50.673789978027344, -19.966869354248047, 53.59152603149414, -15.953006744384766, -14.295486450195312, -14.637319564819336, 43.07014846801758, -22.836692810058594, 31.489931106567383, 25.93368911743164, -1.6454238891601562, 22.808069229125977, -25.89260482788086, -1.6891428232192993, -0.19427497684955597, -18.416318893432617, -2.823742628097534, 33.89788818359375, -7.390777111053467, -47.43373107910156, -17.710330963134766, 29.08681297302246, -8.29678726196289, -11.888543128967285, 56.989479064941406, -9.537471771240234, -48.304054260253906, 0.9727923274040222, -3.859743118286133, 7.070882797241211, -16.404706954956055, 15.53211498260498, 49.687255859375, 32.51060485839844, 75.6261978149414, 14.900188446044922, 7.073464393615723, 14.18175983428955, 24.511425018310547, 31.80248260498047, -9.009953498840332, -21.53959846496582, 22.364181518554688, -44.508602142333984, 9.928747177124023, 25.226993560791016, 2.0056753158569336, 11.257006645202637, 70.1283187866211, 9.179450988769531, -47.006290435791016, 15.253156661987305, -31.96201515197754, -17.210689544677734, 47.46111297607422, -69.40888214111328, 14.73011302947998, 8.167119979858398, -10.720189094543457, 33.17061233520508, -22.966459274291992, -10.069233894348145, -45.32100296020508, -18.3112850189209, -4.248414039611816, -5.265834808349609, -51.4758186340332, 64.48259735107422, -23.653257369995117, -5.59247350692749, 43.54995346069336, -23.347694396972656, -14.57154369354248, -2.407102584838867, -4.077227592468262, 40.042484283447266, -12.038540840148926, 1.3125097751617432, 23.240053176879883, 18.011764526367188, -68.31300354003906, 4.963899612426758, -14.839163780212402, -9.255939483642578, -8.54971981048584, 9.319842338562012, 6.362132549285889, 21.763744354248047, 10.01269817352295, -14.943424224853516, 23.365856170654297, 9.05884075164795, -32.670230865478516, 11.143643379211426, -24.454357147216797, -58.65312576293945, 3.027705430984497, -5.42111349105835, -1.1127946376800537, 4.297435283660889, 18.883981704711914, -13.661504745483398, -5.96181058883667, 47.28202438354492, -55.049137115478516, 25.999143600463867, 16.33049774169922, 14.373147964477539, 23.57328987121582, -42.46037673950195, 33.49600601196289, 26.131832122802734, 36.746517181396484, -0.5673267841339111, -51.89766311645508, -14.255694389343262, 47.785308837890625, 19.784950256347656, 26.088735580444336, 0.34887605905532837, 25.00135612487793, 24.130998611450195, -23.188899993896484, 9.562722206115723, 35.69188690185547, 20.871488571166992, -2.1964309215545654, -10.137617111206055, -5.6484270095825195, -32.48054504394531, -10.339534759521484, 13.445059776306152, 9.369863510131836, -23.734487533569336, -11.001729965209961, -52.569583892822266, -18.58075714111328, -21.735191345214844, 40.75014114379883, -4.098123550415039, -35.96812438964844, -2.5482850074768066, -32.862003326416016, 14.225768089294434, -27.309450149536133, 32.69084548950195, -34.962162017822266, 57.58122253417969, 36.44951248168945, -3.605774164199829, 15.658191680908203, 24.71558952331543, -37.20151138305664, -39.90061950683594, -19.635181427001953, 4.766036510467529, 32.00551223754883, -14.990996360778809, -50.12371063232422, 12.588106155395508, -17.909223556518555, -12.91336441040039, -16.761573791503906, 23.51870346069336, 25.597198486328125, -20.806964874267578, -7.933569431304932, 30.62320327758789, 19.298831939697266, -24.675783157348633, 6.313121795654297, -19.553457260131836, -27.54489517211914, -23.084117889404297, 19.56269645690918, -15.493639945983887, -5.031558513641357, -8.68282699584961, 51.85029220581055, 13.743868827819824, 30.41813087463379, 22.99040412902832, -52.86671829223633, -50.830726623535156, -13.28972339630127, 31.889829635620117, -23.829734802246094, 59.304298400878906, -27.826887130737305, 10.883368492126465, 30.622840881347656, 19.982234954833984, 77.98928833007812, -30.958667755126953, -23.59141731262207, -33.44614028930664, -3.224405527114868, 10.339764595031738, -33.65568923950195, -5.858484745025635, -36.20606231689453, -5.575503349304199, -39.23971939086914, -5.438347816467285, 21.873828887939453, 82.51457214355469, 40.32841873168945, 55.46860122680664, -18.306394577026367, 10.045022964477539, -40.808677673339844, 4.791191101074219, 5.104540824890137, -0.8228926062583923, -9.378936767578125, -7.305605411529541, -42.54861068725586, 23.641923904418945, -46.86289978027344, 38.445289611816406, 71.96778869628906, 19.288095474243164, 36.44455337524414, 32.3155632019043, 3.660183906555176, -24.591873168945312, 40.97127914428711, 20.686756134033203, -40.890045166015625, 70.67385864257812, 0.014852280728518963, -5.7806243896484375, 41.767250061035156, -33.800907135009766, 9.62693977355957, -7.205297470092773, 32.43208312988281, 28.515172958374023, 1.5236032009124756, 18.796409606933594, -22.371444702148438, 48.829837799072266, -12.45767593383789, 67.38809204101562, 43.305999755859375, -14.462925910949707, 2.836930513381958, -13.766511917114258, -13.642468452453613, -37.7308235168457, 34.31315231323242, 1.6345024108886719, 39.28974151611328, -41.742698669433594, 11.600479125976562, 8.809516906738281, -6.7548956871032715, -9.875459671020508, -40.323143005371094, -31.873157501220703, 1.9386177062988281, 20.09193229675293, -33.0425910949707, 25.06690788269043, 43.051753997802734, 57.46297836303711, 2.0389716625213623, 10.697454452514648, -5.741622447967529, 15.03405475616455, 10.021332740783691, 30.219837188720703, -1.441963791847229, 69.81533813476562, 83.6357650756836, 15.523902893066406, 13.246949195861816, -81.9027099609375, -19.606077194213867, 9.931013107299805, -12.556830406188965, -40.015281677246094, -17.648433685302734, -46.62965393066406, -25.59070587158203, -40.70171356201172, 25.92169952392578, -19.651193618774414, 14.78175163269043, -25.03478240966797, -79.54840087890625, 33.46810531616211, -12.465985298156738, -52.95978927612305, 15.985353469848633, 7.76948881149292, 47.33852005004883, -62.63087463378906, 19.183698654174805, 18.953927993774414, 50.883460998535156, 5.9662017822265625, 0.6031677722930908, 11.766396522521973, 29.66937255859375, -22.627147674560547, -10.209744453430176, 1.2400885820388794, 63.63177490234375, 24.296899795532227, -12.990428924560547, -11.753168106079102, 21.93496322631836, -4.745670318603516, 0.06102314591407776, 38.08406066894531, 5.150546550750732, -35.17605209350586, -6.500702381134033, -21.677419662475586, -63.99468231201172, 27.031648635864258, -6.671382427215576, -36.87000274658203, -8.349325180053711, 41.06977462768555, 18.806119918823242, -51.6038703918457, -36.29918670654297, -13.586725234985352, -19.70257568359375, -19.651710510253906, 9.179460525512695, -42.63637161254883, 16.949892044067383, -7.639351844787598, -79.80423736572266, -21.94220542907715, 36.04952621459961, -20.506637573242188, 13.84533977508545, -23.20392417907715, -18.761484146118164, 40.05614471435547, -11.782441139221191, 39.186676025390625, -40.53515625, -38.721282958984375, 21.566076278686523, -19.2418155670166, -57.87388229370117, -9.706790924072266, 17.333646774291992, -31.063289642333984, -11.888381958007812, -22.767257690429688, 3.2213590145111084, 25.971446990966797, 39.16470718383789, 14.561017036437988, 52.41391372680664, 31.24909019470215, 26.056812286376953, 86.17639923095703, 16.408613204956055, -60.74676513671875, 5.3292436599731445, 15.691036224365234, 46.490665435791016, -7.101078033447266, -41.746116638183594, 46.44097137451172, -54.105804443359375, -20.01738166809082, 0.011552270501852036, -7.168363571166992, -18.816707611083984, -4.652993202209473, -7.364826202392578, -1.8654122352600098, -0.7682944536209106, 9.975096702575684, -44.86517333984375, 18.96434211730957, -16.100162506103516, 48.31466293334961, 15.899941444396973, 12.768794059753418, 27.936195373535156, 45.030975341796875, -23.699596405029297, 61.88849639892578, -27.521656036376953, -4.195315837860107, -79.12174987792969, -4.252496242523193, -18.162622451782227, -37.535240173339844, 16.962862014770508, -2.964449882507324, 26.386934280395508, 2.075937032699585, -8.427107810974121, -55.38042068481445, -15.685243606567383, 51.81473159790039, 2.1930150985717773, 14.302489280700684, -24.820539474487305, -26.42443084716797, -83.21851348876953, -4.494571685791016, 31.73873519897461, -3.6529297828674316, 13.664482116699219, 47.86878204345703, 21.7349796295166, 17.059341430664062, -13.564146995544434, -41.60594177246094, 24.65789794921875, 16.716201782226562, 13.178154945373535, 38.393714904785156, 43.304534912109375, -18.30006217956543, 54.99641799926758, 17.928009033203125, 47.589759826660156, 13.780293464660645, -40.23203659057617, -13.030360221862793, -27.214405059814453, -31.40787124633789, -12.679611206054688, -5.726140975952148, -37.848228454589844, 24.5239315032959, -37.82001495361328, 1.4315061569213867, 22.666425704956055, -30.211790084838867, 4.012096405029297, -5.11149263381958, 10.94179916381836, -22.748123168945312, -15.884233474731445, -3.561979293823242, -65.5300064086914, 42.9383430480957, -1.7697714567184448, -14.639999389648438, -11.587142944335938, -1.8859539031982422, -11.788902282714844, 26.75797462463379, 21.980960845947266, -29.758176803588867, 68.50032043457031, -10.32271957397461, 10.592888832092285, 10.28652572631836, -24.5618896484375, -26.34357261657715, 15.09030532836914, -17.124374389648438, -29.698654174804688, 11.221843719482422, -12.074923515319824, -26.740018844604492, -43.0714111328125, 44.891422271728516, 5.454561233520508, 7.958214282989502, -20.230871200561523, -33.839332580566406, -19.408254623413086, -28.347797393798828, 45.64394760131836, 36.38899230957031, -30.771774291992188, 6.628986358642578, 1.2171738147735596, -9.874935150146484, 17.974952697753906, 54.03767395019531, -42.00003433227539, 56.17031478881836, -31.00433921813965, 24.634323120117188, -61.55354690551758, -38.45283126831055, -24.75714111328125, -10.989919662475586, -30.92645263671875, 39.98800277709961, 0.487754225730896, -3.825500249862671, -21.731922149658203, -14.618042945861816, 50.13736343383789, 16.81494140625, -40.442230224609375, -16.30359649658203, -2.9026238918304443, 0.23342418670654297, -76.09452056884766, 16.274625778198242, -4.627618789672852, 13.40230655670166, 26.51276206970215, -61.88188934326172, -1.821163296699524, 6.487848281860352, 23.86573600769043, -16.104686737060547, 40.79509353637695, -26.291587829589844, -24.57186508178711, 8.503047943115234, -7.1639533042907715, 16.153528213500977, -11.13856315612793, 26.43958282470703, -21.140077590942383, 61.981014251708984, 2.571436882019043, 14.22370719909668, 31.266483306884766, 29.453397750854492, -27.509441375732422, 35.26240539550781, -40.223594665527344, -50.220516204833984, 9.031966209411621, 38.68076705932617, 17.362043380737305, 15.392396926879883, 69.27210235595703, 29.60207176208496, 31.70913314819336, 0.2961108684539795, 24.332765579223633, -25.221487045288086, -3.782606601715088, -20.33944320678711, 65.01025390625, -40.85593795776367, -10.459638595581055, -19.189699172973633, -2.193683624267578, 25.615985870361328, -19.940284729003906, -16.423954010009766, -41.593875885009766, -10.252331733703613, -27.855337142944336, -20.22170639038086, 38.59956741333008, 34.072872161865234, -25.257427215576172, 2.7717487812042236, -50.90419006347656, 18.665117263793945, -19.411340713500977, 2.3104248046875, -5.647684574127197, 44.6516227722168, 5.000247001647949, 31.436466217041016, 1.4126298427581787, 37.393394470214844, -2.479459524154663, -13.989992141723633], [13.803778648376465, -27.249969482421875, -26.158369064331055, 14.563693046569824, -35.65599060058594, 6.057995319366455, 11.790639877319336, -67.2518310546875, -0.4598647356033325, 39.15045928955078, -33.73769760131836, -4.697315216064453, 58.96818161010742, -17.90369415283203, 15.60769271850586, -28.24003028869629, 42.322635650634766, -46.013206481933594, -68.79375457763672, 22.016422271728516, -20.776948928833008, -60.620887756347656, 35.70297622680664, 11.648590087890625, -20.325702667236328, -17.18281364440918, 22.131895065307617, -31.658315658569336, -21.477128982543945, 12.057127952575684, -16.663745880126953, 1.6528804302215576, -34.73192596435547, -8.872451782226562, 14.248090744018555, 14.873759269714355, 3.0576858520507812, 42.569541931152344, 5.129214286804199, -34.676658630371094, 33.49083709716797, 26.53489112854004, -6.820884704589844, -15.900120735168457, 11.980937957763672, 23.791671752929688, -67.94457244873047, -8.250733375549316, -32.69684982299805, -45.85966491699219, 15.042964935302734, 15.870412826538086, -3.3744659423828125, -5.16365385055542, 53.479881286621094, 18.068246841430664, 11.052136421203613, -14.257732391357422, -15.474722862243652, -14.787760734558105, -2.400484561920166, 24.93601417541504, -6.2489752769470215, -12.156693458557129, -22.771501541137695, -55.74551773071289, 16.65454864501953, -21.4367733001709, -58.795562744140625, -10.152420043945312, 65.57209777832031, -10.003414154052734, 28.077131271362305, 1.0167206525802612, -14.540168762207031, 16.81639289855957, -7.997356414794922, 18.7890682220459, 4.163265705108643, -24.529783248901367, -9.297416687011719, -17.47347068786621, 30.8499698638916, -54.92713165283203, -21.320833206176758, 5.9981818199157715, -29.575183868408203, -25.94822883605957, -38.294254302978516, 7.828373908996582, 1.9503173828125, -70.97137451171875, 2.9093523025512695, -34.67282485961914, -18.970521926879883, -10.499919891357422, -23.15924072265625, -31.75104331970215, -38.91373825073242, 9.334284782409668, 36.08949279785156, 15.666168212890625, -8.416455268859863, 3.8398611545562744, -50.17540740966797, -17.947589874267578, 11.848143577575684, -26.995582580566406, 61.69155502319336, 7.78947639465332, 10.786161422729492, -42.324337005615234, 26.219696044921875, 30.9346981048584, -30.444395065307617, 24.7835693359375, -46.32404708862305, 38.822792053222656, -16.327129364013672, 13.88427734375, 75.98136138916016, -9.26702880859375, 41.10436248779297, 24.5699462890625, 23.873310089111328, -25.265920639038086, 26.102310180664062, -44.04737091064453, 32.94703674316406, 30.704605102539062, 69.88294982910156, 15.739919662475586, -17.02960777282715, 29.568527221679688, -38.182613372802734, -13.739197731018066, 52.87067794799805, 15.028068542480469, -9.800664901733398, -43.770851135253906, 11.78856372833252, 16.402576446533203, 18.092845916748047, 6.1098151206970215, 39.03050994873047, 40.22019958496094, -9.487931251525879, -12.777125358581543, -29.092079162597656, -1.6304932832717896, 2.830477237701416, 0.6012623310089111, 59.769874572753906, 26.682781219482422, 6.969173431396484, -29.210830688476562, -37.08488082885742, -27.70823097229004, -28.066200256347656, -0.5543432235717773, 28.271934509277344, 6.222402095794678, 1.0395750999450684, -23.6453800201416, -4.090047836303711, 13.697012901306152, 12.036969184875488, 62.07686996459961, -1.6764416694641113, 26.106168746948242, -18.370872497558594, -7.426064491271973, 19.1529598236084, 32.35273361206055, -5.663218021392822, 12.573221206665039, 19.884933471679688, 6.375782012939453, -22.562644958496094, 4.079387187957764, -27.060142517089844, -16.187156677246094, -1.0513890981674194, 11.659235000610352, -10.276341438293457, -78.34807586669922, -32.832706451416016, -39.79733657836914, 3.6141929626464844, -0.5065318942070007, 15.724010467529297, -30.220352172851562, 32.0732536315918, 29.48548698425293, -31.69841957092285, 22.246814727783203, -19.49457550048828, 25.438512802124023, -33.35548400878906, 20.992813110351562, -28.93828773498535, 36.53114318847656, 6.3250885009765625, -14.014609336853027, -5.550633430480957, -15.683719635009766, 8.946144104003906, -5.738548278808594, -10.220024108886719, -10.17568588256836, -51.52053451538086, -55.6241455078125, -0.04755818843841553, -34.673099517822266, -41.42608642578125, 5.426720142364502, 31.704618453979492, 43.006717681884766, 7.341189384460449, -32.231807708740234, 42.59718704223633, -37.20394515991211, -33.32536697387695, 8.29372787475586, 5.108477592468262, -5.881350994110107, 11.200295448303223, 22.698402404785156, -21.429479598999023, -19.8456974029541, 69.29119110107422, 33.57999038696289, -0.660982608795166, -59.660099029541016, -30.621437072753906, -6.120534896850586, -2.0542337894439697, 28.5228271484375, -12.035343170166016, -0.49692511558532715, 8.25218677520752, -25.27113914489746, 0.9851725101470947, -31.524932861328125, 16.27152442932129, -42.49296569824219, -51.68348693847656, -44.09849548339844, 1.7585535049438477, 4.845312595367432, 28.006458282470703, -12.883861541748047, 48.36849594116211, 7.106451988220215, -22.68446922302246, -34.4013557434082, 49.751258850097656, 56.43330764770508, -7.978439807891846, -33.750789642333984, 21.055959701538086, -22.587871551513672, -35.84918975830078, -65.22821044921875, -4.667412281036377, -36.6800651550293, 2.0979127883911133, -47.8028678894043, 20.369033813476562, -30.509183883666992, 31.074583053588867, 33.56757354736328, 18.959688186645508, 17.93551254272461, -15.264862060546875, -8.313880920410156, -71.97278594970703, -12.534438133239746, 1.9949326515197754, 64.08699798583984, -0.9825447797775269, 2.0773134231567383, 22.218896865844727, -29.077543258666992, -7.342645168304443, -9.017877578735352, -75.88652038574219, -23.000734329223633, -9.655858039855957, 33.624813079833984, 7.2754950523376465, 33.40462112426758, 20.328861236572266, 25.435697555541992, -64.85657501220703, 87.30670928955078, -6.795102596282959, 0.11209368705749512, 29.0897274017334, -26.9066219329834, 4.456663608551025, 16.622282028198242, -19.683517456054688, -4.86333703994751, -21.916471481323242, -13.49937915802002, 36.181522369384766, -53.43771743774414, 21.868915557861328, -26.366975784301758, 54.37443542480469, 32.392784118652344, 16.944782257080078, -10.227300643920898, -27.873531341552734, -74.92989349365234, -14.807530403137207, -15.123136520385742, -2.5077202320098877, 3.067528009414673, 17.165130615234375, -3.161931276321411, 4.966540813446045, -24.711833953857422, -59.30375671386719, -42.92487335205078, -43.13985824584961, 27.12900161743164, -35.10944747924805, 44.914772033691406, -7.469285488128662, -19.189790725708008, 6.1534929275512695, -29.798662185668945, -19.9464168548584, -15.896232604980469, -7.837379455566406, 20.070804595947266, 41.46852493286133, 6.427475452423096, 43.49115753173828, -3.0416479110717773, 28.855562210083008, 13.375648498535156, -33.177581787109375, 19.445175170898438, 9.57290267944336, -16.38801383972168, 4.545408725738525, -14.730608940124512, -3.8556301593780518, -5.949584484100342, 8.400984764099121, -8.966468811035156, 10.396993637084961, -51.73444366455078, 14.167217254638672, 29.505748748779297, -23.521751403808594, 16.48577308654785, -31.04418182373047, -26.249509811401367, 12.694822311401367, 17.02977180480957, -10.015491485595703, 9.342476844787598, 4.431153297424316, 21.16761589050293, 14.062471389770508, 22.174453735351562, 40.22504425048828, 27.574827194213867, 17.37148094177246, -47.53421401977539, 6.567087650299072, -37.23248291015625, 17.224626541137695, -49.3580322265625, 40.11439514160156, -33.38075256347656, 41.44084167480469, -12.711392402648926, 3.6994917392730713, 40.49947738647461, 6.526083946228027, 11.680163383483887, 1.1396620273590088, -33.86745071411133, -54.55155944824219, -24.482135772705078, 23.549577713012695, 15.630254745483398, 71.10227966308594, 47.38261795043945, 35.49309158325195, -48.26250457763672, 38.32044219970703, -53.96341323852539, -72.83893585205078, 50.013954162597656, -15.011630058288574, 43.016685485839844, -22.89424705505371, 50.8615608215332, 0.7715392112731934, 20.559734344482422, -53.728515625, -17.409887313842773, 11.663070678710938, 6.061114311218262, -19.32242202758789, 37.732242584228516, -25.018781661987305, 14.867246627807617, 15.450907707214355, 15.257369995117188, 20.608308792114258, 38.46516799926758, 2.2909460067749023, -30.4791316986084, 3.6996426582336426, 20.039579391479492, 26.656490325927734, 10.199649810791016, 28.732629776000977, -9.942018508911133, -16.900402069091797, 10.674238204956055, -29.990161895751953, 19.00782585144043, -50.31916046142578, 5.414922714233398, -26.49374008178711, -29.607234954833984, -63.780548095703125, -16.15636444091797, -43.070716857910156, -7.363608360290527, 13.159222602844238, -6.462162494659424, 33.0760383605957, 39.40714645385742, 7.3595871925354, 60.5986328125, -1.1721763610839844, 6.116065502166748, -30.314592361450195, -21.106115341186523, 12.536422729492188, -18.313438415527344, -25.712846755981445, 15.268392562866211, -17.137422561645508, -23.92853355407715, -28.797311782836914, -24.469097137451172, 17.35841178894043, 19.477766036987305, 19.294572830200195, 8.718223571777344, 11.77016544342041, 3.8499433994293213, -12.987568855285645, 62.91877746582031, 1.6799343824386597, 20.994108200073242, -36.72129821777344, -11.414124488830566, 23.4107666015625, 49.582454681396484, 21.94628143310547, 18.95557975769043, 25.373069763183594, -29.909730911254883, 4.091946601867676, 23.02790641784668, 8.37439250946045, -7.72472620010376, 4.284457683563232, 72.38373565673828, -35.3056755065918, 24.572546005249023, 32.418888092041016, -7.128020286560059, 18.5118350982666, 35.81779098510742, 32.29743194580078, 45.45447540283203, -53.509952545166016, 16.52606964111328, -13.29726505279541, -31.161338806152344, 34.19145584106445, -23.471206665039062, -0.45403745770454407, 48.77396011352539, -35.64164733886719, 7.1803998947143555, 88.00611877441406, 10.07397174835205, 1.19826078414917, 53.939754486083984, 39.26084899902344, -8.248991012573242, -43.05417251586914, -10.031160354614258, 3.628537893295288, -28.087387084960938, -2.2078821659088135, -24.863739013671875, 22.83730697631836, -25.967723846435547, -15.871637344360352, 1.6243668794631958, -43.111412048339844, -32.850502014160156, -48.19419860839844, -14.63460636138916, 24.70384407043457, -9.212367057800293, 10.483245849609375, 17.52931022644043, 58.87103271484375, 15.148287773132324, 34.09771728515625, -18.931781768798828, 32.19074630737305, 29.71676254272461, 18.269676208496094, -28.697898864746094, -17.564491271972656, 55.21519470214844, -20.208251953125, -2.6879711151123047, 7.886847019195557, 48.70294952392578, 9.596515655517578, 10.74672794342041, 38.0683708190918, -26.457914352416992, 6.360671520233154, -1.7185200452804565, 16.341197967529297, 48.687191009521484, 63.19621276855469, -39.401763916015625, 22.431913375854492, -1.0449306964874268, -24.245885848999023, -22.09119987487793, -63.64674758911133, 45.88170623779297, -26.05912208557129, -38.1701774597168, 18.040376663208008, 34.053592681884766, -8.163993835449219, -22.37557601928711, -8.147014617919922, -65.647705078125, 21.31029510498047, -19.128631591796875, 11.84932804107666, 25.410045623779297, 10.795344352722168, 11.374035835266113, -2.8612351417541504, -69.55815887451172, 25.539276123046875, 53.75173568725586, 2.962350845336914, -22.679119110107422, 6.915853977203369, -29.467853546142578, 45.84782791137695, 37.222076416015625, 32.862945556640625, 13.570748329162598, 41.70245361328125, -4.198529243469238, 41.844913482666016, -19.655414581298828, -16.54361343383789, 3.2475976943969727, 23.8782958984375, -10.678394317626953, 34.65854263305664, -5.908156871795654, -23.48278045654297, 66.76322937011719, -22.18959617614746, 1.517763376235962, -29.75276756286621, 34.617156982421875, 8.978470802307129, 20.14656639099121, -23.25381851196289, -40.61000061035156, -15.101072311401367, -5.026896953582764, -24.631057739257812, 13.06044864654541, -27.994298934936523, -17.404743194580078, -8.68144416809082, 34.858131408691406, 5.135449409484863, 19.95438575744629, 3.866079092025757, 0.5585629940032959, -14.38817310333252, -24.960552215576172, 24.72187614440918, -23.52669334411621, -32.625267028808594, 15.357522010803223, -32.281917572021484, 16.650617599487305, 7.28170919418335, 13.301493644714355, 24.587535858154297, -13.343574523925781, 16.011545181274414, 4.963382244110107, 19.438032150268555, -26.109216690063477, 15.65117359161377, 40.32018280029297, -11.929171562194824, -1.0616405010223389, -28.094196319580078, -21.087533950805664, -21.901172637939453, -6.201406955718994, -39.35982131958008, 42.94025421142578, 0.08038485050201416, 44.17754364013672, -13.965423583984375, -32.522239685058594, -35.00082778930664, -22.888561248779297, 17.69546127319336, 21.2491455078125, -30.801025390625, 16.07347297668457, -29.18888282775879, 25.06011962890625, 32.47826385498047, 37.11427307128906, -33.797821044921875, 8.390777587890625, 22.412538528442383, 34.61049270629883, -60.609371185302734, -2.7869248390197754, -5.356745719909668, -18.626070022583008, 32.61147689819336, -23.266414642333984, 30.339683532714844, -26.470003128051758, -44.554283142089844, 20.21584701538086, 18.75700569152832, -11.737218856811523, -17.34183120727539, -27.9162540435791, 31.91831398010254, -55.7165641784668, -55.284523010253906, -22.57324981689453, -17.038543701171875, 32.91568374633789, -5.283658504486084, -6.2211689949035645, -40.54650115966797, 7.260001182556152, -15.185791015625, -1.899645209312439, 23.42997932434082, -8.319287300109863, -20.248991012573242, 12.546185493469238, -19.655868530273438, 36.6485481262207, 11.253326416015625, -0.6839499473571777, -10.611491203308105, 24.87933921813965, -8.85018253326416, -6.917815208435059, -6.159521102905273, -19.70355987548828, -50.891990661621094, -12.358460426330566, -57.95232009887695, 17.836532592773438, 80.00863647460938, -3.809704065322876, -2.2481274604797363, -17.573877334594727, 52.32737731933594, -24.240047454833984, 4.2206830978393555, -12.473977088928223, 9.566312789916992, 13.562100410461426, 25.27632713317871, -32.41899108886719, -1.8246674537658691, -23.802513122558594, 28.583337783813477, -18.887205123901367, 10.991018295288086, -14.636507987976074, 25.474348068237305, -22.69204330444336, -2.3562123775482178, -48.88608169555664, -13.351287841796875, 36.88827133178711, -34.15687561035156, 41.79698181152344, -9.042139053344727, -22.998485565185547, -6.493992328643799, -17.746341705322266, 23.911518096923828, 34.08027267456055, -5.408918857574463, -25.648569107055664, -35.56708526611328, 10.177665710449219, -23.342275619506836, 17.03226661682129, 11.22012710571289, 14.781760215759277, -7.926072120666504, 15.707460403442383, 29.914525985717773, 31.358097076416016, -0.9667462110519409, -6.933958530426025, -18.360910415649414, 17.364652633666992, 17.295425415039062, -16.845172882080078, 59.4193115234375, 7.270936012268066, 32.506595611572266, -33.3222541809082, -14.92725944519043, 34.115230560302734, -21.698261260986328, -20.852893829345703, -15.966218948364258, 4.623824119567871, -14.793375015258789, -1.5104005336761475, 44.743011474609375, -21.559370040893555, 10.20242691040039, 10.942282676696777, -35.27287292480469, -25.46891212463379, 23.709733963012695, 26.372039794921875, 43.110374450683594, 2.557779312133789, -7.059139251708984, -24.12948989868164, -1.740188479423523, 9.233148574829102, 20.7116641998291, 2.0252113342285156, -7.3408589363098145, 32.45967102050781, -12.78753662109375, 4.220864295959473, 22.392353057861328, -29.444339752197266, 17.09819793701172, -60.16603088378906, 4.581775188446045, -27.66156005859375, -18.057025909423828, -4.903776168823242, 23.477441787719727, 59.27617263793945, -12.350921630859375, -33.4349479675293, 40.53559494018555, 13.595601081848145, -9.877739906311035, 6.127713680267334, 6.917649269104004, 11.589879989624023, -21.275068283081055, 47.6127815246582, 38.00492477416992, 49.261905670166016, -7.964371204376221, -12.947556495666504, 29.798795700073242, 87.4522705078125, 10.215601921081543, 17.612688064575195, 33.625648498535156, -5.210378170013428, 32.90263748168945, -2.0113391876220703, 22.854127883911133, -56.08882141113281, 26.714433670043945, -8.269387245178223, -28.677391052246094, 39.24991226196289, -44.514671325683594, 13.586661338806152, 21.140596389770508, -5.1527276039123535, 100.963623046875, -8.033984184265137, 28.61433219909668, -3.5842509269714355, -16.518047332763672, 17.345029830932617, 3.8770649433135986, -23.27010154724121, 26.48895263671875, -18.562904357910156, -26.840862274169922, -16.613311767578125, 36.66461181640625, -34.1796989440918, -1.4416112899780273, 3.661238431930542, -70.29292297363281, -3.5131239891052246, -8.21779727935791, -37.10136795043945, -29.83661460876465, 7.631373405456543, -15.441699981689453, -39.566184997558594, 38.490516662597656, 29.559303283691406, -12.197004318237305, 28.419628143310547, 3.795045852661133, 24.513916015625, -62.317420959472656, -26.519237518310547, -1.9476852416992188, 53.88055419921875, 2.5327954292297363, 17.035442352294922, -32.232967376708984, -21.84311866760254, 32.85127639770508, 24.475080490112305, 11.18182373046875, 1.0923911333084106, -20.075538635253906, 4.6883769035339355, -17.595895767211914, -38.82249450683594, 35.68868637084961, 2.876556634902954, 54.621482849121094, 14.679380416870117, 3.4544878005981445, 41.87369918823242, -19.66393280029297, 3.3925228118896484, -2.367891550064087, -17.94847869873047, -12.462441444396973, 31.0078182220459, 9.35821533203125, 31.169763565063477, -17.419353485107422, 13.436491012573242, 17.741275787353516, -10.208194732666016, 59.358638763427734, -0.07047158479690552, -25.80158042907715, -32.31110382080078, -53.446006774902344, -8.17125129699707, -15.977194786071777, 39.85081100463867, 49.85575866699219, -1.4287611246109009, 12.374783515930176, 17.63340187072754, 21.195571899414062, -6.079080581665039, 27.678668975830078, -41.91985321044922, -65.98812866210938, 15.350687980651855, -7.713855266571045, 45.827632904052734, 1.2593393325805664, 23.07595443725586, 9.660990715026855, 19.89202880859375, 11.998431205749512, -7.731057643890381, -8.328832626342773, 30.376724243164062, 20.91005516052246, -46.93586730957031, -55.60749816894531, -36.05686569213867, -6.001192092895508, 75.1331558227539, -22.57365608215332, 25.71678352355957, -55.55756378173828, -10.107061386108398, 57.079036712646484, -0.5596334934234619, 22.751911163330078, -43.029666900634766, 24.117122650146484, 15.597890853881836, -38.515621185302734, 4.382984161376953, -11.956558227539062, 44.943946838378906, -11.597291946411133, 35.23517990112305, 29.009010314941406, -47.49642562866211, 21.124006271362305, -2.8245537281036377, 1.3299330472946167, -4.321361064910889, -41.61473083496094, -23.42561912536621, 23.81675910949707, 53.31496047973633, 25.319246292114258, 1.3523460626602173, 13.50767993927002, -15.8840970993042, 18.082599639892578, -26.76671600341797, -40.410057067871094, 43.10361862182617, -12.66240406036377, -38.60901641845703, -15.124825477600098, 3.9147839546203613, 3.304563045501709, 33.07546615600586, 23.487773895263672, -12.038994789123535, -37.96046829223633, 18.57589340209961, -55.99620056152344, 17.236034393310547, -4.130803108215332, -10.996830940246582, -44.654598236083984, -27.981966018676758, 44.65141677856445, -14.272467613220215, 7.2624311447143555, 6.972796440124512, -17.660850524902344, -8.857565879821777, 36.6668586730957, -47.99287414550781, 22.235769271850586, 36.25147247314453, -49.73302459716797, -68.47334289550781, 5.707912921905518, 1.7437951564788818, 13.794511795043945, 2.9183013439178467, -46.9534797668457, 9.884641647338867, 19.24161720275879, -8.732233047485352, -0.6772673726081848, 16.86503791809082, 7.838679790496826, 50.313724517822266, -10.839566230773926, 33.90909957885742, -48.11006546020508, 55.92283630371094, 50.595001220703125, -23.858064651489258, -32.72591018676758, -4.290236473083496, 0.5885276794433594, 1.4915611743927002, -44.803646087646484, 3.933176040649414, -18.36807632446289, -53.26945495605469, -12.620709419250488, 44.25274658203125, 17.575176239013672, -33.2365608215332, -37.26877975463867, -29.556346893310547, -15.139150619506836, -10.044029235839844, -16.541549682617188, -26.55301284790039, 11.931145668029785, 1.1318416595458984, 15.459258079528809, 36.78927230834961, -0.37034863233566284, 9.874122619628906, -15.627269744873047, -10.889296531677246, 12.631795883178711, -3.434056043624878, 34.31106185913086, -11.423728942871094, 2.82348370552063, -23.116069793701172, -10.775782585144043, -53.24870681762695, -23.487319946289062, -39.21714401245117, -8.1751070022583, -27.31989288330078, 4.268310070037842, -1.9837003946304321, 1.0867043733596802, -33.41360855102539, -34.872528076171875, 38.15205764770508, -6.246218204498291, 25.48921775817871, -12.276175498962402, 76.12653350830078, -3.0647876262664795, 19.967113494873047, -35.98689651489258, -8.937992095947266, 25.736373901367188, 13.95950698852539, 48.32835388183594, 33.69212341308594, -4.740923881530762, 3.8199732303619385, -43.87881851196289, 2.06358003616333, -33.5304069519043, -17.00183868408203, -29.221446990966797, 96.5404052734375, -37.81056594848633, -18.82204246520996, 126.21601104736328, 18.102203369140625, -32.485172271728516, -11.257763862609863, -13.393817901611328, 32.44662857055664, -3.4903366565704346, 11.540498733520508, 30.235395431518555, 62.190956115722656, 4.301702976226807, 30.8414249420166, -4.6499505043029785, -19.533987045288086, -29.739648818969727, -3.3184585571289062, 20.988840103149414, 18.222156524658203, 22.46408462524414, -48.59545135498047, -69.61170196533203, 46.12209701538086, 48.22352600097656, 27.538888931274414, -11.662833213806152, 10.635836601257324, 36.2132453918457, 48.004676818847656, 28.59964370727539, 2.6065354347229004, 18.314258575439453, 1.0127460956573486, -21.074378967285156, -27.75621795654297, 0.40793484449386597, 22.676767349243164, -11.634864807128906, 11.051609992980957, -17.485910415649414, -1.8594560623168945, 21.527402877807617, 10.66285514831543, 40.76668167114258, 3.188441753387451, 21.15009307861328, -13.44387435913086, -14.279387474060059, 5.008815765380859, -16.162622451782227, 60.85357666015625, 25.889575958251953, -13.639041900634766, -14.970565795898438, 13.60748291015625, 9.270159721374512, -16.8369083404541, -82.20935821533203, -10.464037895202637, -29.571115493774414, -14.095579147338867, 19.54413414001465, 1.904252052307129, 25.059873580932617, -9.829690933227539, -27.033414840698242, 14.873047828674316, -36.88969802856445, -0.30154937505722046, 12.055768966674805, 20.046401977539062, -35.740047454833984, -2.2979536056518555, -6.447794437408447, -9.190728187561035, 13.275491714477539, 29.43694496154785, -14.16207504272461, -27.997913360595703, 27.710676193237305, -10.061845779418945, -34.630592346191406, 3.632223129272461, -34.19964599609375, -1.3392562866210938, -5.903624057769775, -8.078815460205078, 38.6113395690918, -19.714818954467773, -25.08389663696289, 69.93478393554688, 3.3894035816192627, -6.447610855102539, 0.3913542628288269, -31.432649612426758, -2.7191951274871826, -22.66449737548828, 13.454485893249512, -12.1862211227417, -34.03685760498047, -33.22871780395508, -13.033738136291504, 29.79718017578125, 3.0666396617889404, 32.12326431274414, -8.942119598388672, -31.63252067565918, 3.9983901977539062, -41.278255462646484, -12.520833969116211, 34.25245666503906, 47.6436653137207, -21.67227554321289, -25.55621910095215, -20.4547176361084, -3.3608686923980713, -20.614795684814453, 40.09320068359375, -33.380916595458984, -18.676626205444336, -25.736953735351562, -6.071710586547852, 35.85482406616211, -31.903974533081055, 21.465553283691406, -29.789323806762695, 18.59510040283203, 1.071068525314331, 1.2619327306747437, 1.9335777759552002, 7.92385196685791, -8.930158615112305, 35.66482925415039, -30.715850830078125, 27.972204208374023, 4.667731761932373, 20.993873596191406, 25.330322265625, -20.42935562133789, 15.274765014648438, 8.816370010375977, 11.952205657958984, 13.26172924041748, 3.2219276428222656, 62.001651763916016, 55.99386978149414, -0.37554681301116943, 32.2449836730957, 36.789424896240234, 12.537604331970215, -36.590476989746094, 58.199031829833984, 32.345542907714844, -0.48381340503692627, -18.81261444091797, -13.968575477600098, -3.4877939224243164, 8.193221092224121, 16.863950729370117, 25.209304809570312, 68.61251831054688, 10.345945358276367, -14.47408390045166, -13.252241134643555, 19.05486488342285, 38.367897033691406, -38.9307861328125, 70.683837890625, 27.747949600219727, 22.03744888305664, 15.2379150390625, -15.89909839630127, -10.827508926391602, 2.844851016998291, 20.24384880065918, 21.72624397277832, 22.927215576171875, -27.716073989868164, -12.844757080078125, -7.236457824707031, 58.76888656616211, -45.01835250854492, 14.682239532470703, -29.60795021057129, 54.451419830322266, -23.93967056274414, -15.127396583557129, -21.730926513671875, 23.209644317626953, 5.770303249359131, -21.425037384033203, -26.258506774902344, -25.94933319091797, 37.074546813964844, 12.50606632232666, 37.55262756347656, -11.578384399414062, -11.245351791381836, -10.217028617858887, -12.699173927307129, -36.59329605102539, -12.577045440673828, -58.98117446899414, 8.95731258392334, -6.847223281860352, 22.980138778686523, 52.54909896850586, 35.006404876708984, 49.71080017089844, 25.959659576416016, 19.912715911865234, 11.847890853881836, 48.53609848022461, -26.15311050415039, 7.3144707679748535, -37.92654037475586, 37.73042678833008, -15.481114387512207, -13.734574317932129, -11.370542526245117, 2.1501293182373047, -14.496788024902344, 15.961915969848633, 36.64955520629883, -31.25594711303711, 59.62733840942383, 5.937184810638428, -21.98299217224121, 7.44540548324585, 18.06611442565918, 31.965951919555664, -4.514324188232422, 17.08294105529785, -30.165552139282227, 13.513824462890625, -13.70616340637207, 31.773237228393555, 78.28502655029297, -55.007568359375, 29.897079467773438, 78.17298126220703, -9.40079116821289, -29.824356079101562, 22.06433868408203, 2.2064154148101807, -13.585047721862793, 29.54280662536621, -6.521396160125732, 32.929779052734375, 12.401033401489258, 6.492806911468506, -28.96640396118164, -31.317094802856445, -23.521406173706055, 24.798095703125, -17.19171905517578, 29.3408260345459, 15.17541217803955, 20.46442985534668, 57.72508239746094, -25.33405876159668, 12.266091346740723, -7.431898593902588, 35.971290588378906, -21.18466567993164, -65.29883575439453, -22.241878509521484, 17.380586624145508, 9.95484447479248, 31.331066131591797, 30.550329208374023, 38.51137924194336, 3.921140670776367, -33.136783599853516, -17.438276290893555, 17.323083877563477, -46.17477035522461, 14.950313568115234, -17.95880699157715, 8.215873718261719, 64.87277221679688, -27.41511344909668, -16.845653533935547, 46.54795455932617, 0.4691939949989319, 12.709795951843262, 8.768718719482422, 47.502376556396484, 18.58890724182129, 0.9137063026428223, -36.80869674682617, 13.821146011352539, -23.241254806518555, -71.88320922851562, -69.31809997558594, -44.127281188964844, 91.94696807861328, 24.62850570678711, 13.873847007751465, 5.930973529815674, -7.650275230407715, -20.052549362182617, -75.5280990600586, -7.5881476402282715, 18.855451583862305, -20.244667053222656, 21.24395179748535, -1.315519094467163, -19.863018035888672, -45.5753288269043, -22.971511840820312, -4.417853832244873, -18.484577178955078, -18.998798370361328, -48.058475494384766, -44.0042839050293, 35.752342224121094, 8.130220413208008, -9.456809997558594, -11.211752891540527, 24.804576873779297, 59.26083755493164, -28.959266662597656, 8.719205856323242, 42.99592971801758, 12.564909934997559, -41.44847106933594, 56.1550407409668, -33.90584945678711, -18.0606689453125, 36.13594055175781, -13.490680694580078, 35.04914093017578, 41.25678634643555, -3.5458128452301025, -6.789428234100342, 46.79629898071289, -2.3028459548950195, -27.86014175415039, -6.778770446777344, 7.427403450012207, -7.400965213775635, 8.286609649658203, 8.442705154418945, -36.46784973144531, -34.64153289794922, 1.626197099685669, 31.46668243408203, -78.71448516845703, -4.34979772567749, 13.527111053466797, 21.839736938476562, 14.302597045898438, -28.09761619567871, 17.779407501220703, -27.431547164916992, -28.091318130493164, -21.595579147338867, 35.86582565307617, 18.551456451416016, 40.79911422729492, 27.580745697021484, 1.4392151832580566, -13.932591438293457, -47.39552688598633, -26.06941032409668, 17.4348087310791, 43.172340393066406, 10.719240188598633, 57.564754486083984, -14.38972282409668, 60.40632247924805, 4.22258186340332, 28.706758499145508, -15.434341430664062, -12.18349838256836, 4.044375896453857, 28.78915023803711, 26.241790771484375, 9.615123748779297, -38.16740798950195, 22.31077003479004, -7.373229026794434, -19.438732147216797, -60.581661224365234, 48.44633102416992, 53.21792984008789, -39.25975036621094, 9.408981323242188, -29.89646339416504, -3.587054967880249, -28.98294448852539, 25.68897247314453, -6.952747821807861, -2.150012493133545, 14.516898155212402, -31.741092681884766, -103.55947875976562, 48.67253875732422, 4.899094581604004, 3.2772035598754883, -1.0978469848632812, -3.0516421794891357, -18.260684967041016, -57.8781852722168, -26.565767288208008, -12.080312728881836, -7.244526386260986, 13.437783241271973, 34.152862548828125, -0.07992018759250641, 17.607343673706055, 39.55567932128906, 61.226295471191406, -18.356792449951172, -5.2056884765625, 21.272884368896484, -47.32928466796875, -23.61495018005371, -11.35308837890625, -36.91464614868164, 32.5581169128418, 17.417688369750977, 77.87201690673828, -5.7989654541015625, 33.05369186401367, 9.953563690185547, 45.14741516113281, 49.64394760131836, -16.785728454589844, -26.338497161865234, -2.424534320831299, -13.4066162109375, 15.255746841430664, 10.402571678161621, 18.17399024963379, -45.18251419067383, 19.305696487426758, -11.954163551330566, -28.72785186767578, 16.71563148498535, 38.28908920288086, -11.085542678833008, 25.167953491210938, 14.441760063171387, 13.130611419677734, 17.04648208618164, -22.770578384399414, -27.877952575683594, 39.09103012084961, 13.917594909667969, 23.837764739990234, -46.003700256347656, -9.497908592224121, 53.001338958740234, 44.96107482910156, 17.250913619995117, -10.159320831298828, -27.51674461364746, -14.387551307678223, -46.578468322753906, -27.243656158447266, -26.79617691040039, 34.63493347167969, -5.400693416595459, 19.907203674316406, -13.89979076385498, 28.368972778320312, -38.827938079833984, -20.858325958251953, 27.276365280151367, -20.448251724243164, -5.674983501434326, -0.9799854755401611, 24.75806427001953, -9.11180591583252, -43.817176818847656, 23.506017684936523, 50.09591293334961, -41.5657958984375, 45.67719268798828, 58.18760299682617, -38.68572235107422, 38.126426696777344, 22.076871871948242, -1.3515045642852783, -3.014853000640869, 11.479852676391602, -23.41121482849121, -3.6023597717285156, -27.394168853759766, 45.989158630371094, 2.816120147705078, -14.396723747253418, 11.983346939086914, 1.6483497619628906, 4.140103816986084, -24.546388626098633, -1.2369788885116577, -1.3650386333465576, 54.37986373901367, -6.522319316864014, -54.16598892211914, -36.44667434692383, -19.013551712036133, 16.228294372558594, 48.39748001098633, 10.072293281555176, 57.48891830444336, -22.969093322753906, -1.0275802612304688, 37.35063934326172, 5.941260814666748, 27.274524688720703, 39.41633987426758, -9.632973670959473, -1.6486265659332275, -17.57647705078125, -44.9992561340332, 52.78076934814453, -40.953189849853516, 22.54608154296875, -22.3501033782959, -10.985227584838867, -7.642287254333496, 18.274280548095703, -21.939990997314453, -34.923404693603516, 18.498916625976562, -20.45069694519043, 21.382261276245117, -26.670133590698242, -19.175233840942383, -40.371299743652344, -33.91362380981445, 33.32270050048828, -6.339630126953125, 14.45517635345459, 22.593320846557617, -14.438283920288086, -7.685430526733398, 11.802279472351074, 11.360179901123047, 89.00537109375, 3.3025002479553223, -6.781228065490723, -53.199424743652344, -37.09327697753906, 23.7828311920166, -9.447051048278809, -22.725566864013672, 18.003293991088867, 29.15839958190918, -37.35309600830078, -17.076377868652344, 22.06833839416504, 23.624752044677734, -7.583536148071289, -8.493361473083496, -5.639683723449707, 26.45002555847168, -35.479949951171875, 62.949554443359375, 25.77798080444336, -52.13833236694336, -0.5176509618759155, 20.741811752319336, -10.073629379272461, 7.147224426269531, -60.6970100402832, 53.70757293701172, -25.658512115478516, -18.00984764099121, -59.21416091918945, 20.259082794189453, -17.384653091430664, -3.8991217613220215, 48.97268295288086, -51.135520935058594, -0.07948887348175049, -55.42915725708008, 18.002622604370117, 6.951007843017578, 2.6307339668273926, 6.26730489730835, -19.60634994506836, 6.7345170974731445, 6.305174350738525, -5.937636375427246, 26.67711067199707, 16.242040634155273, -8.115373611450195, 7.624484062194824, 32.68000411987305, 38.08218765258789, -0.7076841592788696, -3.361400604248047, 3.9321882724761963, 26.05616569519043, 13.568954467773438, -12.59793472290039, -18.019317626953125, 8.940383911132812, 21.854318618774414, 26.864009857177734, -27.369789123535156, 9.227703094482422, -16.72774314880371, 41.8201789855957, 11.085622787475586, 0.5420801043510437, -10.973052024841309, 12.713664054870605, -5.946763515472412, 26.645732879638672, -34.21556854248047, -26.621641159057617, 6.160037040710449, -3.1273276805877686, -63.01095199584961, 10.409601211547852, -34.610538482666016, -6.293105602264404, -23.546052932739258, 78.49268341064453, -13.415082931518555, 27.927688598632812, -1.618729829788208, -9.205392837524414, 27.56438446044922, -43.68772888183594, -62.510589599609375, -4.897769927978516, -29.340970993041992, -38.33156204223633, -26.311616897583008, 0.5729506015777588, -0.8746104836463928, -33.039085388183594, -23.803707122802734, 7.347067356109619, 17.556177139282227, 8.553516387939453, 26.8276424407959, -47.35669708251953, 36.24290466308594, -40.975704193115234, -40.44099807739258, 11.59175968170166, 44.65791702270508, -30.214458465576172, -10.767241477966309, -20.006200790405273, -15.330842018127441, 27.851242065429688, -31.670026779174805, 0.4582316279411316, -9.222786903381348, -32.509437561035156, 0.5527660846710205, 23.070789337158203, 29.77910041809082, 12.280949592590332, -43.924930572509766, -41.10649871826172, -26.208478927612305, -26.70509147644043, 4.801127910614014, -41.73151397705078, 30.531030654907227, 60.8285026550293, 6.864640235900879, 28.563186645507812, 13.820879936218262, -58.41885757446289, 44.86967086791992, 27.337129592895508, -49.1562614440918, 2.4013876914978027, 18.80853271484375, 10.36246109008789, 81.2590560913086, 31.243989944458008, 2.83174991607666, -68.80184173583984, -51.049346923828125, 59.762351989746094, -40.11320495605469, -37.8781852722168, 39.2603645324707, -9.439685821533203, -32.740081787109375, 12.586005210876465, 9.185657501220703, 11.610342979431152, -23.09241485595703, 7.481492519378662, 35.97483825683594, 9.206857681274414, -20.11740493774414, 37.10103225708008, -3.910888671875, -41.51034927368164, 20.860578536987305, -17.84674072265625, -22.781885147094727, -3.995180606842041, 28.815513610839844, -37.18067169189453, 47.49568176269531, 52.54285430908203, 20.335474014282227, 2.3875722885131836, -41.32352066040039, 9.85341739654541, 39.40692901611328, -21.536073684692383, 7.67935037612915, 5.937051773071289, 19.102394104003906, -12.749812126159668, -11.131153106689453, 46.72446823120117, -1.5087236166000366, 22.434293746948242, -19.584693908691406, 16.996511459350586, -26.53093719482422, -12.108133316040039, -24.029647827148438, -17.766948699951172, -1.6119542121887207, 7.945061206817627, -10.69943618774414, -1.5887712240219116, 32.841739654541016, -5.070429801940918, 47.69342041015625, -12.903350830078125, 36.04393005371094, -4.837078094482422, 21.144617080688477, 2.969844102859497, 21.99988555908203, 16.872777938842773, -67.86587524414062, -27.861665725708008, -3.2181668281555176, 70.9144515991211, 4.3775200843811035, 44.51815414428711, -37.32865905761719, -24.479446411132812, 41.11665725708008, 16.434553146362305, -1.7765284776687622, -20.71904945373535, -11.860801696777344, -17.49028778076172, 13.843120574951172, -1.851062536239624, -13.265676498413086, 4.552353382110596, 27.350252151489258, -49.18364715576172, 44.99030303955078, 18.299842834472656, 15.42353630065918, -29.68607521057129, 31.12236213684082, -13.0385160446167, 32.04041290283203, -23.981565475463867, -13.784296035766602, 7.4628496170043945, -0.29704588651657104, 4.770203590393066, -36.99753189086914, -16.546072006225586, -29.55377960205078, -48.87379837036133, 17.809667587280273, 36.45139694213867, 13.906840324401855, -26.097490310668945, -18.28367805480957, -7.006353378295898, -14.205743789672852, 52.30302047729492, -0.022039592266082764, -27.25516128540039, 20.149295806884766, -10.116981506347656, -4.190533638000488, -7.431949615478516, 40.656044006347656, 10.086137771606445, 30.030553817749023, -14.435935020446777, 7.275275230407715, -26.765321731567383, -27.480541229248047, -57.69185256958008, 61.3089714050293, -27.80398178100586, -24.567115783691406, -42.757781982421875, 39.32716369628906, -31.584741592407227, -16.523466110229492, -39.63196563720703, 13.10012149810791, 27.72395896911621, -26.686893463134766, 21.667152404785156, 10.289163589477539, 25.10021209716797, -18.279558181762695, -47.660823822021484, 12.59365177154541, 20.137819290161133, -62.61720275878906, 18.462806701660156, 3.131664991378784, 43.282955169677734, 23.876562118530273, -15.677936553955078, -16.558361053466797, -53.273414611816406, -27.219715118408203, -15.612874984741211, 0.18830974400043488, 39.742855072021484, 44.87787628173828, -39.81512451171875, 29.893470764160156, 18.742382049560547, -4.247380256652832, 18.550289154052734, -30.296035766601562, 11.37874698638916, 9.972908973693848, 44.455284118652344, 21.788015365600586, 15.508255958557129, 21.371353149414062, 13.313863754272461, -32.686275482177734, -32.188167572021484, -15.676440238952637, -31.569690704345703, -23.621732711791992, 18.39470863342285, 65.9400863647461, 6.20763635635376, -35.27751922607422, -19.662561416625977, -40.206790924072266, 17.761837005615234, -26.183313369750977, 37.83689498901367, -1.978531002998352, 47.591400146484375, 51.6476936340332, -3.331176280975342, 21.392534255981445, -29.38311004638672, -16.000993728637695, -14.209573745727539, 10.30421257019043, -28.233869552612305, -85.78765106201172, -0.280346155166626, 7.0255022048950195, 6.3884358406066895, 8.070121765136719, -17.34995460510254, 85.99419403076172, -39.767757415771484, -23.311885833740234, 55.79359817504883, 13.509162902832031, -31.172082901000977, 33.862091064453125, -22.617929458618164, -1.7876580953598022, 38.138153076171875, 28.900325775146484, 65.88385772705078, 25.500141143798828, -35.6139030456543, -0.664937436580658, -7.153258323669434, 13.161821365356445, -0.4258871376514435, -28.896461486816406, 23.938199996948242, -4.306914806365967, 0.005543172359466553, -15.930865287780762, -6.460661888122559, -66.50922393798828, -23.184524536132812, -29.722599029541016, 12.626716613769531, 16.019977569580078, 10.819497108459473, 5.741057872772217, -49.352718353271484, 42.37038803100586, -12.108661651611328, 14.328573226928711, -2.7759053707122803, 16.231395721435547, 15.981887817382812, 24.245389938354492, -9.585705757141113, -73.77932739257812, -46.92217254638672, -25.17705535888672, 17.820207595825195, 25.315471649169922, 61.60068893432617, -13.134873390197754, -38.77173614501953, -9.328049659729004, 6.500683307647705, 30.64575958251953, 21.836811065673828, -2.678704023361206, 26.17184829711914, 22.432870864868164, 17.34381675720215, 10.199049949645996, -4.321613311767578, -38.332550048828125, -21.28839111328125, 30.249666213989258, 10.513431549072266, 23.00835418701172, -24.11175537109375, 11.658598899841309, -2.601749897003174, 12.74491024017334, -30.476186752319336, 29.389577865600586, -6.89219856262207, -0.3224753141403198, 10.842108726501465, 10.541193962097168, -11.467365264892578, -16.644309997558594, 62.53891372680664, 14.16402530670166, -23.612627029418945, 1.0543010234832764, 28.451520919799805, 65.139892578125, -6.789592742919922, 42.71608352661133, 39.83310317993164, 32.03191375732422, -28.208845138549805, -5.010546684265137, 1.3700551986694336, 21.89557647705078, 4.958175182342529, -16.872095108032227, -7.862064838409424, -0.44035786390304565, 2.6181697845458984, 39.067893981933594, 22.847675323486328, 38.500640869140625, -29.048351287841797, -41.20752716064453, -2.1275460720062256, -48.385318756103516, -8.940139770507812, -10.045083045959473, 29.562557220458984, -26.304794311523438, -23.373693466186523, 69.6120834350586, 26.834104537963867, -18.692960739135742, -18.882875442504883, 7.103233814239502, -17.60494041442871, 13.83635425567627, -15.010262489318848, 2.8100171089172363, 37.17166519165039, -20.870712280273438, -48.9891471862793, 14.20914363861084, -28.24907112121582, 6.061743259429932, -12.786590576171875, -10.531383514404297, -39.930599212646484, -7.04472017288208, 41.38490295410156, -13.857136726379395, -26.201379776000977, 58.01988983154297, -38.861515045166016, 9.78999137878418, 12.672029495239258, 27.09857177734375, -27.290237426757812, -4.940217971801758, 6.233088493347168, 17.371950149536133, -2.8481333255767822, -26.777421951293945, -39.09690475463867, -39.44094467163086, -31.528942108154297, 22.519447326660156, 37.02787399291992, -49.97377395629883, -14.863360404968262, 21.208227157592773, 9.934142112731934, -47.502685546875, -23.95868682861328, -19.825061798095703, -39.84922409057617, -14.954526901245117, 45.67819595336914, 0.944655179977417, 50.04296112060547, -24.072269439697266, -41.25365447998047, -48.00185012817383, 4.201294898986816, 18.59198760986328, 45.95315933227539, -19.468969345092773, -1.2916474342346191, -83.08746337890625, 20.428226470947266, 15.333329200744629, -14.108723640441895, -29.833303451538086, -14.843683242797852, 26.498069763183594, -39.165584564208984, -21.961450576782227, -3.635922908782959, 23.94445037841797, 47.22925567626953, 38.91508102416992, -36.27159118652344, -0.8576877117156982, -15.595538139343262, 60.11029815673828, -26.023372650146484, 1.4701728820800781, -12.627691268920898, -7.6988043785095215, 12.207326889038086, 4.877565860748291, -11.89229679107666, 40.56492614746094, 33.1279182434082, -8.9893798828125, -45.96638488769531, -15.725028038024902, -4.324402809143066, 9.256428718566895, -17.72838592529297, -7.3892927169799805, -30.45758628845215, -5.285701751708984, -15.345540046691895, 11.318724632263184, 12.273423194885254, 64.22843933105469, 17.311389923095703, 9.079558372497559, 31.722679138183594, 28.330121994018555, -2.585999011993408, -10.29779052734375, 47.43150329589844, -2.1058690547943115, 39.606563568115234, 33.04302215576172, -20.127269744873047, -26.198759078979492, 3.753905773162842, -49.01638412475586, -15.33979606628418, -43.79698181152344, -49.023475646972656, -15.5657320022583, 29.30996322631836, 1.7852253913879395, -53.06035232543945, 3.1829936504364014, 46.3428840637207, -28.342655181884766, 3.0614867210388184, -19.234582901000977, 24.19392204284668, -11.648750305175781, -31.91280174255371, -23.173500061035156, 18.880870819091797, -12.534384727478027, 8.74789047241211, -27.959388732910156, 1.451577067375183, 17.778160095214844, 15.282270431518555, 31.201587677001953, 7.5261640548706055, -13.062121391296387, 2.958163261413574, 18.636137008666992, 2.0518908500671387, 7.285650253295898, 9.151281356811523, 6.814465045928955, -6.561884880065918, -25.083524703979492, -4.529324054718018, -34.833648681640625, 32.05881118774414, -6.855084419250488, -17.24053955078125, -17.51495361328125, -50.38071823120117, 13.65108585357666, 45.60627365112305, -50.748390197753906, 2.271169662475586, -5.150448322296143, -11.417726516723633, 23.535676956176758, 4.977232456207275, 5.319243431091309, 46.764949798583984, 47.232322692871094, 7.8776044845581055, -20.51378631591797, 17.71938705444336, 9.389039039611816, -28.417098999023438, -47.916412353515625, -17.248971939086914, 64.82258605957031, -28.051422119140625, 44.348880767822266, 34.85762023925781, 1.01227605342865, -54.577754974365234, 67.59307861328125, 17.751998901367188, -3.7761073112487793, 29.17488670349121, -14.728736877441406, -1.1837000846862793, -27.386003494262695, 10.359334945678711, -3.515664577484131, 39.27424621582031, 25.2154598236084, 35.31771469116211, -11.141985893249512, 2.617733955383301, 19.884693145751953, -60.851131439208984, -5.021393299102783, 15.630849838256836, 35.626953125, -39.85437774658203, 4.193188667297363, 41.35842514038086, 10.515884399414062, -64.15156555175781, 11.277353286743164, -47.5689697265625, -15.153512001037598, -24.939476013183594, -22.8189697265625, 26.679136276245117, 40.91411590576172, -10.745349884033203, -9.526224136352539, 48.91517639160156, -28.766372680664062, 23.798574447631836, -30.99744987487793, 22.721588134765625, -18.18279266357422, -55.40142822265625, -40.669090270996094, 14.678495407104492, -43.63026809692383, 9.46821403503418, -53.61387252807617, -9.027090072631836, -56.54180145263672, -29.226268768310547, 2.159097671508789, 35.93359375, 40.96141052246094, 42.32941436767578, 64.32124328613281, 22.618131637573242, 6.308039665222168, -36.48245620727539, 15.537680625915527, -14.725330352783203, 37.26264190673828, 59.84684753417969, -8.35464859008789, -53.596614837646484, -32.65400314331055, 3.895998001098633, -0.5856021642684937, -33.699954986572266, 32.433868408203125, 3.1140310764312744, -0.30336225032806396, -6.074141502380371, -30.53618621826172, 1.098777413368225, 59.58266067504883, 40.2316780090332, 16.65557098388672, 55.456974029541016, 1.7755646705627441, 9.592696189880371, -14.864213943481445, 13.941248893737793, 44.48146438598633, -49.151615142822266, 4.426609992980957, 20.54530143737793, -37.12873077392578, 8.98433780670166, 1.9000993967056274, -31.91267204284668, -29.460145950317383, 34.01095199584961, -33.89973449707031, 6.767254829406738, 6.4675211906433105, -41.54087448120117, -62.75294876098633, -23.500732421875, 23.636390686035156, 5.0069732666015625, -0.6649299263954163, 12.604050636291504, 4.9065680503845215, 6.499594211578369, -4.866289138793945, 16.739469528198242, 12.236824035644531, 20.72385597229004, 25.92793083190918, 1.7700079679489136, 45.22077178955078, 8.641229629516602, -14.293330192565918, -100.65171813964844, 66.39192962646484, -18.963451385498047, 1.9206434488296509, 13.741491317749023, 49.5765380859375, 32.17727279663086, 17.053667068481445, 32.94025802612305, 3.449209213256836, 24.50320816040039, -30.01374053955078, 5.632669448852539, -26.18094825744629, 70.8127212524414, -16.05860710144043, 40.177005767822266, -29.74872589111328, -48.639198303222656, 49.73597717285156, 7.748785018920898, -31.172985076904297, -44.14147186279297, -5.452282428741455, 2.8012278079986572, 22.17205238342285, -50.50031280517578, 17.226408004760742, -9.842702865600586, -25.504150390625, -24.60134506225586, -1.9033788442611694, -0.8939831256866455, 2.6603968143463135, 30.5690975189209, -57.322879791259766, 28.31786346435547, 18.21627426147461, -18.739744186401367, -13.791360855102539, 6.519824028015137, 81.30998229980469, 2.616459369659424, -63.903053283691406, 12.470771789550781, 33.05622482299805, 25.548751831054688, -39.8220100402832, 4.010880470275879, 39.3803825378418, 26.805843353271484, -0.9749523997306824, -3.8291211128234863, 21.104074478149414, -6.6291303634643555, -30.652265548706055, -30.17623519897461, 54.28886413574219, 19.834455490112305, 30.22440528869629, -5.4759745597839355, -1.5156810283660889, -47.23740005493164, 11.465873718261719, 5.325026512145996, 15.829086303710938, -13.542196273803711, -19.594411849975586, 29.993043899536133, -14.955032348632812, -9.23520278930664, -23.538944244384766, 12.410009384155273, 23.872024536132812, 6.057981014251709, 1.7414367198944092, -76.95144653320312, 26.25248146057129, -46.12042999267578, 0.5250529050827026, -36.21816635131836, 53.760013580322266, 20.229686737060547, 25.714399337768555, 12.111759185791016, 16.618370056152344, 65.95028686523438, 3.506411552429199, -65.59770965576172, -23.239940643310547, 45.5745735168457, -17.93244171142578, -19.03178596496582, 36.54949951171875, -19.972122192382812, -26.69662094116211, -4.226734161376953, -34.82490539550781, -5.905890464782715, -38.51408386230469, 18.36778450012207, 28.426387786865234, 32.307655334472656, 23.763458251953125, -13.280123710632324, -22.60401725769043, -28.332191467285156, 20.635452270507812, -4.229611873626709, -18.66741371154785, 9.273799896240234, -21.207801818847656, 6.362997531890869, 10.692239761352539, 26.728681564331055, 28.673295974731445, -46.9201774597168, -21.534198760986328, -5.589910984039307, -43.258026123046875, 30.320892333984375, 22.74782371520996, 1.9215145111083984, 27.037837982177734, -26.881591796875, -0.14438772201538086, 22.400907516479492, -15.487611770629883, -39.75940704345703, 10.300556182861328, -41.12201690673828, -1.2744688987731934, 14.677230834960938, 13.879186630249023, 15.659795761108398, -3.235970973968506, 49.656166076660156, 9.697628021240234, -16.726268768310547, -12.510750770568848, 10.42299747467041, -40.15281677246094, -11.262775421142578, 26.824304580688477, 31.536026000976562, -24.475156784057617, -4.2492995262146, 29.355066299438477, 29.876863479614258, -53.17505645751953, 3.5532259941101074, 7.596165180206299, -0.5952906608581543, 23.542831420898438, -15.240568161010742, -8.30187702178955, 9.383642196655273, 22.833114624023438, 12.142030715942383, -34.21498107910156, -23.63199234008789, 6.969361305236816, 11.184924125671387, -20.685989379882812, 17.968849182128906, 62.77555847167969, -22.597492218017578, -2.476929187774658, 35.00682830810547, -15.225529670715332, -3.1787796020507812, -7.507739543914795, -6.121490001678467, -9.215276718139648, -8.553184509277344, -12.66499137878418, -13.59322738647461, -6.506417274475098, -83.87371063232422, -83.30694580078125, -11.091851234436035, -3.6184091567993164, 30.562122344970703, 6.448427677154541, 13.505515098571777, -40.64281463623047, 13.87889575958252, -23.486478805541992, -56.58102035522461, 35.213905334472656, 72.8956527709961, 42.62751770019531, 9.81238842010498, -58.2781867980957, 1.4693371057510376, 22.631765365600586, -19.213918685913086, -11.401701927185059, 31.296865463256836, -13.224185943603516, -61.4490852355957, 32.22393035888672, 0.3909170627593994, -9.138784408569336, 8.603355407714844, -9.030332565307617, -8.387264251708984, 17.714902877807617, -29.019329071044922, -45.184268951416016, 6.102558612823486, -30.659788131713867, -17.60586929321289, 3.432051420211792, -26.07994270324707, 43.979244232177734, -9.341134071350098, -4.759460926055908, -42.33561706542969, -34.12161636352539, 54.184417724609375, 16.870603561401367, -21.74700164794922, -12.174079895019531, -23.59734344482422, -12.605484962463379, -34.705352783203125, -21.818763732910156, 7.235462665557861, -10.384683609008789, -26.99188232421875, 2.0486292839050293, -84.27810668945312, -12.99367618560791, 27.003053665161133, 14.623983383178711, -4.76984977722168, -0.7880438566207886, -2.343841552734375, 1.6378002166748047, 11.33464241027832, -8.38908576965332, 45.051185607910156, 4.771626949310303, -42.206241607666016, 25.300443649291992, -69.18708801269531, -26.14779281616211, 18.25267791748047, -60.340248107910156, -22.80992317199707, 22.447717666625977, 22.844520568847656, -0.3372284770011902, 14.898104667663574, 30.3474063873291, 16.182430267333984, 6.779374122619629, 18.954736709594727, -23.865585327148438, -0.49834853410720825, -15.533308982849121, -15.876503944396973, -3.5903077125549316, -48.70289993286133, -23.188180923461914, 1.9202673435211182, -23.355030059814453, -27.98736000061035, 21.38477897644043, 0.38308948278427124, 21.342945098876953, 16.976104736328125, 69.08975219726562, -5.8989577293396, 7.462362766265869, -4.5924882888793945, 0.879501461982727, -25.89437484741211, 8.87698745727539, -37.12065124511719, 5.447041988372803, 7.644423007965088, 49.13581848144531, -65.16661834716797, 12.62568187713623, -52.73201370239258, -2.7421295642852783, 15.871166229248047, -9.577037811279297, -16.024551391601562, 0.674560010433197, -27.77073860168457, 64.18047332763672, 44.04206466674805, 36.77894973754883, -11.259288787841797, 53.196102142333984, 31.147645950317383, 21.521390914916992, -24.980792999267578, 32.307090759277344, -24.985815048217773, 61.549964904785156, -19.16170883178711, -27.46112060546875, 39.05192184448242, 12.75499439239502, -19.078968048095703, -0.1910184621810913, -23.86448860168457, 7.214519500732422, 17.30034637451172, 30.173534393310547, -13.06132698059082, -10.416234970092773, 25.799556732177734, -24.205039978027344, 19.88313102722168, 6.919801712036133, 1.6775705814361572, 44.089786529541016, -30.654560089111328, -1.0584355592727661, 49.16452407836914, -14.36297607421875, 13.35922622680664, -18.90213394165039, 29.909032821655273, 6.189947128295898, -1.5665299892425537, 5.878339767456055, -13.360870361328125, -3.7381160259246826, 6.020491123199463, 37.88109588623047, -59.531490325927734, 49.10205841064453, -28.763742446899414, -30.802410125732422, 31.1401309967041, 46.06583786010742, 4.015809059143066, -13.327438354492188, 20.833126068115234, 15.613077163696289, 10.600028038024902, 12.263751983642578, -20.28080177307129, -27.00789451599121, 3.789088726043701, -23.295494079589844, -33.73942565917969, -30.55423355102539, 17.888723373413086, -57.88670349121094, 13.17874813079834, -76.44410705566406, -54.13606262207031, -2.903277635574341, -10.344232559204102, -6.089443206787109, 33.975101470947266, 27.195772171020508, -1.9922525882720947, 14.926753044128418, 13.865056037902832, -35.004600524902344, 22.37240219116211, -37.004520416259766, 32.176361083984375, -11.391517639160156, -32.69643020629883, 48.131107330322266, 8.537991523742676, 3.644474506378174, 8.602140426635742, 21.779823303222656, 46.6839599609375, -20.925926208496094, -18.962894439697266, -60.36454772949219, -17.074260711669922, -33.90898513793945, -38.0115966796875, -11.295883178710938, 50.35652542114258, 4.160901069641113, -2.8992207050323486, 21.12885093688965, 3.6575987339019775, -17.3346004486084, -7.886654376983643, 16.378433227539062, 25.242450714111328, -50.59121322631836, -7.124673366546631, 51.73200988769531, -20.547758102416992, 22.895889282226562, -49.48527526855469, 13.828824043273926, -14.414546966552734, -24.498186111450195, 3.1993775367736816, -16.169124603271484, -1.9312368631362915, -66.8040542602539, 24.14695167541504, 2.609042167663574, 37.14592361450195, -30.31146240234375, 0.8512952327728271, -35.81459045410156, -12.91073226928711, 38.18815612792969, 16.52459144592285, 26.368587493896484, -5.118027687072754, -51.654563903808594, 4.980152130126953, 72.95067596435547, -15.140531539916992, -4.603820323944092, 9.74050521850586, 7.530589580535889, 55.89375686645508, -19.422866821289062, -43.95854187011719, 14.709575653076172, 3.722342014312744, -2.8683061599731445, 16.589941024780273, -20.137615203857422, 6.2663798332214355, 57.47538375854492, -16.624624252319336, -22.224946975708008, -23.36501693725586, 45.152099609375, -7.42424201965332, -34.68394470214844, 17.95981788635254, -60.52227020263672, 24.882741928100586, -20.705053329467773, -18.305368423461914, -8.37056827545166, 64.75981903076172, -1.1081933975219727, 0.41216814517974854, -3.1859490871429443, -44.989681243896484, 13.096969604492188, -30.5622501373291, -14.045299530029297, -13.335749626159668, -35.00517272949219, -12.170110702514648, -35.19453430175781, 11.578638076782227, 18.45020866394043, 33.69550704956055, -66.02983093261719, -7.145824432373047, -18.72411346435547, 11.371027946472168, 10.696220397949219, 48.87002944946289, 22.81403350830078, -5.808062553405762, 27.199893951416016, -29.22194480895996, 16.171363830566406, -7.916836738586426, 15.554388046264648, 50.50757598876953, -38.78157043457031, 47.67943572998047, 38.93659210205078, 17.828845977783203, -22.591659545898438, -30.76387596130371, 26.69294548034668, -12.550541877746582, 26.505346298217773, -33.11083984375, 9.756511688232422, 27.550724029541016, -67.87606811523438, 35.15805435180664, -16.954246520996094, -45.77266311645508, 44.132381439208984, -37.3128547668457, -15.998867988586426, 17.99407386779785, -17.718334197998047, -9.608895301818848, -22.660808563232422, 10.560979843139648, 40.185752868652344, -6.813124179840088, 0.16231726109981537, -17.273387908935547, 10.558145523071289, -96.84088134765625, -10.99215316772461, 7.080845832824707, -21.64933204650879, 28.409862518310547, -10.959881782531738, -23.46477508544922, 11.412095069885254, -12.086793899536133, 8.205842018127441, -30.048921585083008, -0.5638039708137512, -12.560037612915039, 4.419742584228516, 46.447513580322266, -24.749502182006836, 31.852907180786133, 23.299007415771484, 29.167959213256836, -25.26401138305664, -49.17904281616211, -31.877443313598633, -10.505434036254883, -35.282535552978516, 33.412174224853516, -7.681007385253906, 49.0627326965332, 49.41275405883789, 15.812167167663574, -34.980857849121094, 26.956872940063477, -24.41526985168457, 30.377777099609375, -31.997495651245117, 6.528602123260498, 11.988861083984375, 46.73843765258789, -62.51361083984375, 35.12080383300781, -4.210011005401611, -7.499701499938965, -45.898292541503906, -30.057689666748047, -27.263893127441406, 36.722801208496094, 14.096328735351562, -37.71400833129883, 51.842201232910156, 1.2228398323059082, -34.503082275390625, -3.418123722076416, -9.73408317565918, 26.619611740112305, -20.9913387298584, 50.0138053894043, -35.95859909057617, 55.94357681274414, -16.3891544342041, -24.60515594482422, -11.69321060180664, 37.287384033203125, 39.172576904296875, -16.627626419067383, 37.25708770751953, -44.21823501586914, -14.871417999267578, 16.832515716552734, 6.926318168640137, -7.765239715576172, -29.969188690185547, -34.168575286865234, 6.7198286056518555, -10.301785469055176, -3.8393893241882324, 3.57853364944458, -55.58226013183594, 23.148548126220703, 9.151442527770996, -51.33008575439453, 27.373638153076172, 26.609407424926758, -2.4004788398742676, 9.308051109313965, 5.805576324462891, -15.787107467651367, 16.007959365844727, 27.93009376525879, 17.918968200683594, -6.0380401611328125, 21.33626937866211, 20.579166412353516, -19.565528869628906, -18.140535354614258, 2.9089534282684326, -14.85541820526123, 5.53319787979126, -16.822114944458008, -19.526214599609375, -2.941114664077759, -1.1791986227035522, 3.714378833770752, -41.065059661865234, -4.910430908203125, 6.151867389678955, 15.521859169006348, -1.2796295881271362, -18.774721145629883, 10.84506607055664, -15.197630882263184, 61.800838470458984, 85.99463653564453, -41.69194412231445, -37.17991638183594, 18.87142562866211, -27.510421752929688, 58.55548858642578, 10.750819206237793, 4.4862589836120605, -58.400238037109375, 27.847087860107422, 0.9100483655929565, 33.279293060302734, 13.06184196472168, 0.5347970724105835, -49.29924011230469, -11.043609619140625, 17.391876220703125, 32.721412658691406, -16.543474197387695, -2.5284805297851562, -9.93592643737793, -24.4981746673584, 15.172810554504395, -49.45664596557617, 20.152584075927734, -67.29142761230469, -6.797749996185303, -34.08408737182617, -5.83099365234375, -7.206201076507568, 28.910972595214844, 7.042031764984131, 48.074893951416016, -12.680167198181152, -8.95440673828125, 12.039111137390137, 10.28188705444336, -43.99951934814453, 52.13886642456055, 19.66579246520996, 7.668619632720947, -1.797000765800476, 0.661385178565979, 3.437727212905884, 33.3743782043457, -15.709936141967773, -4.3018693923950195, 59.755218505859375, 4.053014755249023, -15.318204879760742, -45.34224319458008, 19.216625213623047, -33.39702224731445, 17.185632705688477, 34.7037239074707, -4.782465934753418, -23.08083724975586, 30.162288665771484, -0.3656766712665558, -26.87516975402832, -34.337738037109375, 11.60759449005127, -34.60965347290039, 3.7709083557128906, -15.460001945495605, -12.790170669555664, -12.79182243347168, 14.927968978881836, -14.661758422851562, 20.27218246459961, 47.59121322631836, 6.705254554748535, 60.741580963134766, -18.836233139038086, -30.729345321655273, 27.85405921936035, 28.06285858154297, -17.66871452331543, -14.54790210723877, 2.466794729232788, -7.81558895111084, 50.52553939819336, -37.983177185058594, -2.90590238571167, -38.23921585083008, -10.293318748474121, 35.1181755065918, -12.735207557678223, 33.12788772583008, 34.43115234375, 18.01261329650879, 6.706384658813477, -45.559268951416016, 25.37269401550293, 39.839111328125, -45.21830749511719, 40.76726531982422, 22.769123077392578, 32.79963684082031, 18.456146240234375, -19.830074310302734, -33.364952087402344, -11.754619598388672, 27.09868621826172, 4.170217037200928, -57.38182830810547, 11.770098686218262, -32.568782806396484, 23.398406982421875, 28.487651824951172, -4.386394023895264, 26.9924373626709, -28.913721084594727, -21.660654067993164, -6.400096893310547, 83.6416015625, 46.34758758544922, -35.59576416015625, 16.535030364990234, 45.9139404296875, 16.888111114501953, -1.1734671592712402, -17.132387161254883, -21.03754425048828, -19.825000762939453, 18.87527847290039, -53.949440002441406, 35.42140197753906, 22.688962936401367, -54.95213317871094, 17.59189796447754, -12.6616849899292, 26.937402725219727, 20.31075668334961, -66.36262512207031, 6.953804016113281, 10.125555038452148, -47.486724853515625, -10.311715126037598, -24.929115295410156, 51.79746627807617, -16.793903350830078, -9.515957832336426, -15.902313232421875, 5.887753963470459, -5.605564117431641, -35.5854606628418, -31.716127395629883, -15.204380989074707, -50.57865524291992, -11.375716209411621, -28.655277252197266, 32.141849517822266, 21.602867126464844, -100.79095458984375, -5.356308460235596, -25.66936492919922, -27.482013702392578, -51.48003005981445, 12.860122680664062, -0.12470996379852295, -61.74900436401367, 5.637941837310791, -21.147140502929688, 20.86027717590332, -9.438035011291504, 23.840431213378906, 22.60817527770996, -46.494834899902344, 25.231836318969727, 16.899398803710938, 40.13683319091797, -17.7398681640625, 43.0892219543457, -11.928584098815918, -11.053797721862793, 21.463409423828125, 45.66448211669922, -21.023239135742188, -27.89379119873047, 63.54204177856445, 53.07392883300781, 0.8110028505325317, -60.58156967163086, -0.3690476417541504, -9.604853630065918, -15.236776351928711, -20.19146728515625, -42.00299835205078, 27.148649215698242, -21.005464553833008, 47.21292495727539, 13.821925163269043, 27.914091110229492, -4.963595390319824, -38.75478744506836, 7.147461414337158, 41.55133056640625, 28.292438507080078, 18.57007598876953, -6.073695182800293, 49.53256607055664, -42.52999496459961, -17.417766571044922, -5.558097839355469, 6.871556758880615, 1.5936083793640137, -27.03012466430664, -51.02890396118164, 9.712701797485352, 32.82222366333008, 0.494952917098999, -8.731204986572266, -21.349214553833008, -3.569370746612549, 21.79216766357422, 25.005802154541016, 74.17597961425781, -36.416934967041016, 7.467474937438965, -41.41472625732422, 54.97242736816406, -11.94936466217041, -23.51966667175293, 22.0252742767334, -9.801860809326172, -10.133026123046875, -19.25965118408203, 32.844120025634766, -37.73816680908203, -2.364267349243164, -9.328057289123535, -11.372519493103027, 36.237060546875, 21.999204635620117, 5.040915489196777, 29.2086181640625, -2.9883108139038086, -6.723730087280273, 30.72539710998535, -12.25851058959961, 20.4427433013916, 12.76606559753418, 3.79225492477417, -44.695526123046875, 14.352606773376465, -0.7129455804824829, -14.09508228302002, -24.026227951049805, 0.1307561993598938, 5.20259952545166, 10.027422904968262, 43.88737869262695, 42.92852783203125, 14.031537055969238, -23.04971694946289, 1.5404913425445557, 12.133179664611816, -30.327939987182617, -58.13309860229492, -19.294097900390625, -13.115252494812012, -40.820526123046875, -73.88192749023438, 38.65166473388672, 8.141125679016113, -44.64530944824219, 0.7063853740692139, 46.496646881103516, 17.324260711669922, 31.2008056640625, 41.666378021240234, 27.249168395996094, -40.82414245605469, 23.749404907226562, -17.912731170654297, 10.409449577331543, -6.405797004699707, 20.048316955566406, -9.784720420837402, 17.364952087402344, 19.290830612182617, 1.3205029964447021, -7.933125972747803, 8.05335807800293, -12.718896865844727, -9.3101806640625, 15.37785530090332, 0.9199722409248352, -21.540517807006836, -6.650801181793213, 20.632097244262695, 40.7706413269043, -17.913253784179688, 33.367332458496094, -21.31390380859375, -25.113679885864258, 3.021480083465576, -3.5470800399780273, -9.026837348937988, -15.09714412689209, -1.992885947227478, -12.582929611206055, 10.820175170898438, 19.618778228759766, 14.229549407958984, -28.155536651611328, -12.683910369873047, -0.04361259937286377, 0.13882696628570557, -15.547985076904297, 14.451099395751953, 66.96134185791016, -17.693565368652344, -0.795749306678772, -72.50887298583984, -22.241436004638672, -5.326614856719971, -7.242103099822998, -38.237510681152344, -32.716461181640625, 53.62617874145508, -30.425813674926758, -51.27421569824219, -2.9501779079437256, -0.4584769606590271, -26.055265426635742, 62.53076171875, 11.750065803527832, 18.3372745513916, 33.1058349609375, -33.91365051269531, -6.0422797203063965, 3.2524476051330566, 62.97554397583008, -14.362634658813477, -2.6953554153442383, -17.764122009277344, 28.045269012451172, -4.624712944030762, 40.596923828125, 31.247163772583008, 30.25258445739746, -16.59990119934082, -3.067697525024414, 14.055705070495605, -27.752981185913086, -38.83122634887695, 5.4527907371521, -4.277100086212158, 63.48851776123047, 3.5357232093811035, -4.957789897918701, -13.213983535766602, 30.986927032470703, -9.7701997756958, 8.078435897827148, -11.518098831176758, -45.697044372558594, -0.41994932293891907, 28.669130325317383, 5.190627098083496, -6.76767635345459, -50.78391647338867, 19.596527099609375, -23.54856300354004, -28.057308197021484, 54.15049743652344, 13.897391319274902, -10.664098739624023, -7.401375770568848, -14.557684898376465, 45.349979400634766, -37.84396743774414, -1.7846219539642334, 10.771714210510254, -0.8683574795722961, -9.673440933227539, -2.786024808883667, 5.588375091552734, 3.618358850479126, 19.299278259277344, -6.893144130706787, 57.70149230957031, 24.260704040527344, -31.1256103515625, -34.50411605834961, 12.387927055358887, -42.34476852416992, 28.537612915039062, 26.63364601135254, -0.6063730120658875, -62.20026779174805, 4.702836513519287, 74.19840240478516, -16.125553131103516, 24.697011947631836, 11.516371726989746, 4.662140846252441, 14.26279354095459, 29.937589645385742, 3.703942060470581, 18.566577911376953, 0.11026400327682495, -45.8846321105957, 10.72347354888916, -34.65810775756836, -21.855756759643555, -19.303640365600586, -18.981605529785156, -66.56578063964844, 16.65582847595215, 58.54414749145508, -57.684391021728516, 1.8735663890838623, -67.26121520996094, -55.42188262939453, -30.411230087280273, -34.94894790649414, -32.57246780395508, -40.35034942626953, 41.81576919555664, 37.30057907104492, -17.525753021240234, -29.395830154418945, -8.325249671936035, 16.830778121948242, 26.86003875732422, 3.7592339515686035, 19.63294792175293, 22.681167602539062, -22.298547744750977, 48.0299186706543, 12.034159660339355, 13.087793350219727, 3.6910195350646973, 27.13796043395996, 36.07329177856445, 2.753013849258423, -53.20723342895508, 2.6351544857025146, -16.295061111450195, -56.040950775146484, -49.16831970214844, 53.978363037109375, -5.0847368240356445, 3.7398078441619873, -22.0743350982666, 25.790029525756836, 125.16045379638672, -30.831798553466797, 8.074039459228516, 8.629833221435547, -30.33880043029785, -26.453258514404297, -8.342944145202637, 17.42069435119629, -30.34442710876465, 8.336080551147461, -7.066628456115723, 61.03479766845703, -17.371572494506836, 57.93366622924805, 37.95704650878906, -27.6361026763916, -48.650230407714844, 38.367000579833984, 20.272567749023438, -30.820087432861328, 76.40928649902344, -16.430923461914062, 19.598770141601562, -34.09183120727539, -38.85446548461914, 26.828542709350586, 62.52507781982422, 35.8647575378418, 21.49294662475586, -3.1183998584747314, -61.24393844604492, -12.98448371887207, 32.65974426269531, -1.6581147909164429, 7.509586811065674, -48.43610763549805, -2.76200532913208, -13.367898941040039, 9.291403770446777, 3.985278367996216, 36.67533493041992, -29.28101348876953, 57.721439361572266, -34.81142044067383, 43.65108108520508, -18.18971061706543, -34.083133697509766, 10.937962532043457, -51.422515869140625, 12.219581604003906, 33.39393615722656, 28.423959732055664, -34.62335968017578, 10.727246284484863, 11.810369491577148, -19.800207138061523, 29.57542610168457, 14.328994750976562, 65.10079956054688, -26.201906204223633, 8.393547058105469, 23.860462188720703, -19.971912384033203, 19.245298385620117, 6.600764751434326, 52.19456100463867, -8.49440860748291, 11.698965072631836, 32.14663314819336, -27.23418617248535, -12.122848510742188, -16.17385482788086, 38.7586669921875, 6.252508163452148, -6.156698226928711, -7.7671051025390625, 0.059954047203063965, -2.566917896270752, 0.332958459854126, 50.8327522277832, -11.285150527954102, 10.604013442993164, 10.660017013549805, 3.3839821815490723, -1.156812071800232, 4.206368446350098, 36.746192932128906, -10.449230194091797, -7.085595607757568, 0.6609681248664856, -38.09202194213867, -19.622629165649414, 46.131858825683594, -35.031314849853516, -20.014568328857422, 18.722381591796875, 35.74559783935547, 9.180902481079102, -20.71503448486328, -22.32927894592285, 11.527779579162598, -31.191286087036133, -38.3120002746582, 16.955097198486328, -13.25281810760498, -57.819576263427734, 15.352601051330566, 28.907264709472656, 25.41436195373535, 61.61912155151367, 30.33574104309082, -56.21222686767578, 43.81792068481445, 4.0555419921875, 47.19220733642578, -72.3548583984375, -70.9556884765625, 18.32045555114746, -34.455318450927734, -51.065330505371094, -48.735198974609375, 10.329391479492188, 79.92982482910156, -7.230270862579346, -35.49802780151367, 10.705978393554688, 27.703861236572266, 22.824464797973633, 29.72288703918457, -27.887399673461914, 29.24173355102539, -3.5631682872772217, -37.673709869384766, 15.19390869140625, -1.3268522024154663, 18.481138229370117, -38.894798278808594, 1.2202571630477905, 11.419110298156738, 46.19064712524414, 1.8056111335754395, -25.401697158813477, 13.459757804870605, 8.61954116821289, -5.050541400909424, 13.039855003356934, 3.9717273712158203, -40.90242004394531, 40.189754486083984, -18.121315002441406, -2.2364773750305176, 73.11255645751953, -23.2540225982666, 17.494243621826172, 5.460512638092041, -31.804458618164062, 9.434473037719727, -73.53086853027344, 5.95444917678833, -23.029857635498047, 23.435380935668945, 62.83295822143555, 63.9337158203125, 49.639930725097656, -7.559880256652832, -29.849714279174805, -20.35540199279785, -3.4042789936065674, 6.533336162567139, -1.2308119535446167, -3.668196201324463, -2.703312635421753, 1.3995600938796997, -57.56504821777344, -9.67764663696289, 25.113418579101562, -28.872615814208984, 5.670102596282959, -30.726186752319336, 10.92494010925293, 20.526029586791992, 25.64391326904297, 45.0667724609375, -23.751983642578125, -7.760961532592773, 1.223576307296753, -5.9914984703063965, 4.277848720550537, 2.5957672595977783, 0.7974407076835632, -9.590688705444336, 2.5173511505126953, -25.036306381225586, -18.685583114624023, -48.03546905517578, 40.48921203613281, 50.42511749267578, -33.09503173828125, 35.990936279296875, -54.331207275390625, 32.416114807128906, -10.698636054992676, -79.9004898071289, -12.925212860107422, 20.638986587524414, 48.031471252441406, 10.424970626831055, -25.462125778198242, 56.01063537597656, 2.148064136505127, 20.95899772644043, 18.88075065612793, -22.667856216430664, 27.442474365234375, 19.077049255371094, -5.827590465545654, -12.004068374633789, 17.283227920532227, -27.443470001220703, 2.082423686981201, 24.21398162841797, 0.33777618408203125, -13.76389217376709, -8.011141777038574, -40.22304153442383, -28.183090209960938, -44.20115280151367, -44.13642120361328, -41.581886291503906, -35.708984375, -35.27804946899414, 8.057168006896973, -11.459480285644531, -20.31639862060547, 12.016855239868164, 5.374448299407959, -6.367713451385498, 6.739358425140381, -8.068483352661133, 45.13695526123047, 26.96286392211914, -29.705684661865234, -10.910097122192383, 5.032596588134766, 6.750760078430176, 34.82529067993164, -4.977407932281494, 5.834590911865234, 28.29045295715332, -64.06185150146484, -32.22123718261719, 31.417728424072266, 18.62119483947754, -41.5168571472168, -23.419687271118164, -25.575132369995117, -24.570608139038086, 23.894779205322266, 57.728271484375, -32.617618560791016, -40.895057678222656, 43.28718566894531, 37.84185791015625, -39.638484954833984, 59.79726028442383, 10.335552215576172, 16.88829231262207, 65.34774780273438, -32.76499938964844, -25.903867721557617, 27.913068771362305, 27.156959533691406, 16.633176803588867, 108.02079772949219, 6.090648651123047, 49.33037567138672, 18.735626220703125, -4.590773105621338, 0.8346596956253052, 65.31926727294922, -4.373331546783447, -3.753028631210327, -38.69439697265625, -20.462358474731445, -22.90858268737793, 16.712968826293945, -23.264379501342773, -15.88117790222168, -17.08774185180664, -48.09919357299805, -18.685049057006836, 17.81081199645996, 29.798215866088867, 11.191328048706055, 17.749156951904297, -37.77792739868164, -12.406749725341797, -17.985342025756836, 5.091698169708252, 25.332971572875977, -46.68290710449219, 3.14457631111145, -56.271942138671875, 54.70468521118164, 12.00357437133789, -17.518674850463867, 20.25908088684082, -9.102432250976562, 10.07206916809082, -1.3289368152618408, -41.246768951416016, -8.137393951416016, 17.89712142944336, 20.297086715698242, 57.62730026245117, 17.90550422668457, -54.613197326660156, 16.417665481567383, -10.26939868927002, 21.627012252807617, 19.358009338378906, -19.216459274291992, -55.17734909057617, -37.65584945678711, 46.94820785522461, 1.7576017379760742, -28.43755531311035, -4.277948379516602, -7.801355361938477, 58.79216003417969, 11.079182624816895, -18.360267639160156, -19.164594650268555, -33.48686981201172, 6.704178810119629, 41.21268844604492, 9.054738998413086, -9.660663604736328, -17.22328758239746, -5.071825981140137, -9.715376853942871, -35.513694763183594, -38.273136138916016, 38.184871673583984, -18.92316436767578, 15.320148468017578, 61.00780487060547, -21.1639461517334, -16.136144638061523, -17.677921295166016, -55.29115295410156, 80.76844787597656, 28.84563446044922, 7.469598770141602, 21.972801208496094, 38.605464935302734, -41.308074951171875, -29.740116119384766, 11.301909446716309, -1.2476328611373901, -27.885976791381836, -4.60915470123291, -61.491294860839844, -17.216127395629883, 39.09326934814453, 34.21388626098633, 27.40716552734375, 6.870109558105469, -45.218387603759766, -54.584190368652344, 8.697681427001953, 31.361328125, 31.374229431152344, 36.64203643798828, -18.152603149414062, -21.449472427368164, 0.8425713777542114, -41.92319107055664, -39.42532730102539, 12.559814453125, -22.17357063293457, 73.92630004882812, -34.55213928222656, -36.490447998046875, 6.54775857925415, -36.710906982421875, 30.929092407226562, -2.1826257705688477, -28.747461318969727, 0.6206251382827759, -83.25341033935547, -20.989919662475586, -21.447786331176758, 36.82143020629883, 6.261159420013428, 37.72877502441406, -46.1810188293457, 41.085243225097656, -7.238963603973389, -7.7711181640625, 39.90420913696289, 39.27882385253906, 39.191402435302734, -79.67581939697266, -6.406125068664551, 57.90338134765625, -1.30492103099823, -40.805389404296875, -17.600826263427734, -29.410228729248047, -23.19630241394043, 22.739713668823242, 13.70594596862793, 41.57680130004883, 54.95561981201172, -37.40625, 17.904573440551758, 34.841896057128906, -89.30247497558594, 2.8065860271453857, 17.072694778442383, 27.274885177612305, 58.043514251708984, -5.248778820037842, 1.4427497386932373, -0.49625086784362793, 30.8992977142334, 20.946805953979492, -40.931182861328125, -11.998351097106934, 33.691341400146484, -43.01927185058594, 67.4288101196289, 15.39894962310791, 25.286041259765625, -21.03812599182129, -6.708320140838623, 12.538668632507324, -2.8379619121551514, 1.7425661087036133, -9.849318504333496, 1.164775013923645, -36.55547332763672, 20.382856369018555, 4.003851890563965, -9.080406188964844, -0.7395788431167603, 1.2103010416030884, -31.296520233154297, -1.0953108072280884, 22.200138092041016, -10.446930885314941, 9.482730865478516, 13.7868013381958, -23.469810485839844, 19.045509338378906, 23.45199966430664, 41.08384704589844, -9.612926483154297, -8.366435050964355, -29.196210861206055, 26.381750106811523, 12.922972679138184, 8.709025382995605, 18.044004440307617, 26.804290771484375, 41.53468704223633, -3.223123073577881, 47.524166107177734, -4.819485187530518, 64.47544860839844, 0.8997275829315186, 17.48509979248047, 20.344942092895508, -18.05155372619629, -49.93101501464844, 43.85425567626953, 65.45822143554688, 27.495359420776367, 44.36338806152344, 13.98581314086914, -18.727527618408203, -19.074710845947266, 22.83380126953125, 37.002662658691406, -7.120420932769775, -20.436145782470703, -9.213232040405273, 0.663070023059845, -65.50274658203125, 35.449729919433594, -45.785858154296875, -29.315963745117188, 8.40731430053711, -7.790536880493164, -46.19611358642578, -40.73542404174805, -37.08300018310547, 59.993099212646484, -38.07387161254883, -35.46005630493164, 38.99934005737305, 17.376150131225586, 11.368415832519531, -48.27532196044922, -55.988346099853516, -15.409124374389648, -7.8528923988342285, 31.120365142822266, 39.22995376586914, 8.45915412902832, 5.312738418579102, 58.78139877319336, -19.97713279724121, -42.21788787841797, -37.21739959716797, 25.4871826171875, -17.717803955078125, -12.330565452575684, -42.8455924987793, 41.12318801879883, -44.34926986694336, 0.3085590600967407, 8.43397331237793, 17.067174911499023, -23.6572265625, -10.20984172821045, -9.821293830871582, 2.406043291091919, -15.922940254211426, 38.083370208740234, 70.1073226928711, 1.7399519681930542, 11.25434684753418, -2.760267496109009, 52.575584411621094, -28.905675888061523, 27.47556495666504, 7.371013164520264, 22.613229751586914, -18.914798736572266, -16.980810165405273, -18.35688591003418, -13.13301944732666, -2.549098253250122, -65.62284088134766, 8.176947593688965, 13.863455772399902, 20.55596160888672, -26.713043212890625, 16.64865493774414, -16.535804748535156, 7.631618499755859, 18.923189163208008, 27.59332275390625, 59.801963806152344, -17.236337661743164, -22.91688346862793, -50.60728073120117, 0.27333465218544006, 136.00025939941406, -42.36201095581055, -4.606922149658203, 17.380842208862305, -16.73884391784668, 47.13298797607422, -18.338436126708984, 4.864297389984131, -32.32560348510742, -37.677894592285156, 20.354890823364258, 23.667573928833008, -21.053142547607422, 14.99557876586914, 3.534501791000366, 22.03287124633789, 15.290816307067871, -6.390109062194824, 28.553438186645508, -57.61635208129883, -51.557464599609375, -11.407148361206055, 40.676918029785156, 43.510135650634766, -22.435436248779297, -5.785355567932129, -34.388484954833984, -2.9010045528411865, -52.357635498046875, -4.543267726898193, 8.463029861450195, -54.26172637939453, -31.213329315185547, 25.741525650024414, 7.657871246337891, -44.68560028076172, -6.6114044189453125, 23.2845516204834, -40.994869232177734, 14.215208053588867, 4.760643482208252, 2.0541701316833496, 36.075927734375, -26.08522605895996, 12.56152629852295, 40.774932861328125, -4.032558441162109, -12.738075256347656, -13.888534545898438, 10.167562484741211, -7.432481288909912, -22.704118728637695, -20.88111686706543, 28.74854850769043, 9.601581573486328, 49.19104766845703, 49.153831481933594, 43.145694732666016, -13.072545051574707, 11.924072265625, 31.912235260009766, -4.996326446533203, -4.120743751525879, -18.35927963256836, 40.519615173339844, -2.505977153778076, 32.51551055908203, -16.49671173095703, -46.09068298339844, 11.319207191467285, -41.274169921875, -32.25033187866211, -64.87500762939453, -51.825443267822266, -6.482071399688721, -15.387518882751465, -18.416484832763672, 11.831038475036621, 0.5070777535438538, -7.151939868927002, -11.531108856201172, 52.07846450805664, 25.070898056030273, -30.193490982055664, -19.958059310913086, -41.336181640625, -6.222015857696533, -14.55683422088623, -4.202762603759766, 32.85779571533203, -1.6534595489501953, 17.910417556762695, -15.671319007873535, -3.9419734477996826, -13.372041702270508, 40.17100143432617, -28.276199340820312, 59.52880859375, 9.614123344421387, -28.2263126373291, -26.243301391601562, -6.8434014320373535, -10.813031196594238, -1.895995855331421, -8.197352409362793, 20.526798248291016, -56.833927154541016, -16.651813507080078, 13.342092514038086, 0.9034867882728577, -36.95606994628906, -5.081119060516357, -41.089569091796875, -32.339866638183594, 18.00606346130371, -28.543447494506836, 13.331625938415527, 8.541611671447754, 21.408002853393555, -12.69369888305664, -24.249263763427734, -1.9669595956802368, -18.114376068115234, -32.03761291503906, -52.37138366699219, -44.105384826660156, 30.457548141479492, 37.464012145996094, -20.300085067749023, 34.98028564453125, 3.9892265796661377, -7.751416206359863, -26.465604782104492, 6.291985511779785, -0.8030709028244019, 11.8682861328125], [-2.7929372787475586, -23.302165985107422, -15.176956176757812, -33.18790054321289, 18.129440307617188, 57.30087661743164, -31.155323028564453, 32.2115478515625, -11.699836730957031, -12.329615592956543, -29.06471824645996, 8.033263206481934, 25.035375595092773, 15.75765323638916, -15.258660316467285, 18.153135299682617, -36.673744201660156, -38.88328552246094, -14.350610733032227, -0.8831491470336914, -25.18052864074707, 17.84683609008789, -5.6302313804626465, -10.717877388000488, -5.815878391265869, -3.6100730895996094, -45.018226623535156, 8.440875053405762, 72.98311614990234, 13.160512924194336, 6.673408031463623, 12.54113483428955, -4.001688480377197, 0.3915959894657135, 28.814401626586914, -16.283126831054688, 77.17190551757812, 88.30375671386719, 5.077555179595947, 23.361661911010742, -45.495174407958984, 21.310258865356445, -60.29587173461914, -40.47268295288086, 4.08274507522583, -11.679840087890625, -10.524023056030273, 4.948395729064941, 8.472855567932129, 34.682796478271484, 100.56890869140625, -9.925642967224121, -8.381647109985352, 5.062069892883301, 15.759984016418457, 32.01167297363281, 45.159725189208984, 15.607711791992188, 49.946754455566406, 19.96497344970703, -10.012145042419434, -24.854190826416016, -53.974815368652344, -35.6302375793457, -71.01869201660156, 45.900699615478516, 3.2211365699768066, 1.3714818954467773, -6.7226786613464355, -10.784175872802734, -79.97283172607422, 60.144187927246094, 68.52899169921875, -49.49885177612305, -30.126008987426758, -15.056425094604492, 31.37064552307129, 33.571041107177734, 17.431894302368164, 17.922061920166016, 19.385360717773438, -27.610008239746094, 23.722349166870117, 16.879497528076172, 28.681638717651367, 2.118776321411133, -18.7232723236084, 33.59572219848633, 47.92299270629883, 24.044925689697266, 46.2441520690918, -7.681717872619629, -22.012052536010742, -35.352603912353516, 8.842205047607422, -32.61176300048828, 16.85613441467285, 27.053115844726562, -20.006519317626953, -19.6330623626709, 17.12803840637207, 48.18413543701172, 29.299470901489258, -9.108474731445312, 1.2786144018173218, 70.39825439453125, -62.2911262512207, 56.23487854003906, -4.613004207611084, -18.37875747680664, -38.855567932128906, 26.002986907958984, 38.290252685546875, -18.236831665039062, 5.189601898193359, -10.654035568237305, 49.974159240722656, 49.894107818603516, -38.504478454589844, -15.001543998718262, -4.838993072509766, -9.131767272949219, 33.1210823059082, 8.417881965637207, -32.13606262207031, 6.49299430847168, 4.428661346435547, -18.712703704833984, 27.065065383911133, -11.148091316223145, -10.758270263671875, 56.976890563964844, -5.383596420288086, -52.41316604614258, 2.7288262844085693, 29.64068031311035, -1.5426827669143677, -4.820611000061035, -38.83488845825195, 3.124302387237549, -39.910762786865234, -28.223526000976562, 7.901617050170898, 11.664846420288086, 6.81584358215332, -26.568363189697266, 21.463756561279297, 36.28593826293945, -6.257018089294434, -50.52467346191406, -7.189805030822754, -43.622623443603516, 11.672636032104492, 61.041908264160156, -5.6460394859313965, -63.64689636230469, 24.097537994384766, -3.4742379188537598, 31.135242462158203, -35.506778717041016, 34.07878875732422, -27.890657424926758, 1.246253252029419, -33.275272369384766, 12.36257553100586, 28.7121524810791, 0.42104125022888184, 58.24051284790039, 26.08635139465332, -3.9242162704467773, -26.17980194091797, -21.216604232788086, -10.211101531982422, 47.82221221923828, 31.700830459594727, -30.063419342041016, 22.419607162475586, 10.488204956054688, -50.877197265625, -21.834474563598633, 48.59796905517578, 15.290616035461426, -1.0717449188232422, -12.648568153381348, -10.975085258483887, -16.42584800720215, 34.11952590942383, 33.62015151977539, -15.508661270141602, -43.823707580566406, -3.3887453079223633, -18.45940589904785, 19.115234375, -1.0759878158569336, 0.36380279064178467, -2.019436836242676, -30.625221252441406, 27.820220947265625, -44.47573471069336, -20.374399185180664, -17.61518669128418, -41.82667922973633, -47.55494689941406, 20.801694869995117, 83.59920501708984, 29.587326049804688, -15.304302215576172, -31.710525512695312, -6.873928070068359, 41.69456481933594, -9.000040054321289, 39.03997039794922, 44.906864166259766, -19.773191452026367, -22.72346305847168, 46.93360900878906, 17.75367546081543, -19.17841148376465, -1.4025312662124634, -21.95775032043457, 4.05160665512085, 10.097018241882324, -8.05789852142334, -48.87833023071289, -7.163350582122803, 20.0157527923584, -10.55469036102295, -28.849218368530273, -22.669206619262695, 24.30175018310547, -25.895217895507812, -23.88072395324707, -4.772956371307373, 5.6962785720825195, 11.619695663452148, 4.281116485595703, 35.055259704589844, 20.212875366210938, -48.18375015258789, -6.54339075088501, -5.01499605178833, 26.448312759399414, 15.569196701049805, -46.54822540283203, -23.46724510192871, 11.167959213256836, -14.158441543579102, 40.659854888916016, 24.38730239868164, 41.876075744628906, -3.6647214889526367, 50.502681732177734, 44.55207824707031, -6.7170000076293945, 15.545950889587402, 52.866783142089844, 11.056303977966309, 5.358194351196289, 13.974210739135742, 18.87977409362793, -14.11611270904541, -36.944828033447266, -19.126201629638672, -22.290939331054688, 35.52872848510742, -59.83367156982422, -71.32711791992188, -34.99644088745117, 37.04912567138672, -15.970911026000977, -20.786283493041992, -39.09290313720703, -21.84067153930664, -30.54560089111328, 11.68869686126709, -11.868894577026367, -16.37706184387207, 20.45897102355957, 52.93901062011719, -42.29312515258789, -34.92792510986328, 13.980897903442383, 6.634555339813232, 6.114307880401611, -4.110942840576172, 4.495387554168701, 16.506502151489258, -32.424312591552734, 8.778196334838867, -19.64150047302246, -9.838068008422852, -11.75930118560791, -7.924252510070801, 12.2976655960083, 9.365374565124512, -26.725326538085938, -43.92411804199219, 13.627029418945312, 15.122438430786133, 2.2032899856567383, -38.99855041503906, 20.96320152282715, -4.291151523590088, -17.418445587158203, 49.76540756225586, -11.985069274902344, 37.45842361450195, -15.555285453796387, -20.6214656829834, 50.970787048339844, 13.301251411437988, -48.97533416748047, -11.064353942871094, -14.117535591125488, -7.459932327270508, 26.89063835144043, -33.73362350463867, -7.9910969734191895, 18.44841194152832, -5.363630294799805, -20.477672576904297, -0.9287723898887634, 62.35481262207031, 19.161455154418945, -6.282382965087891, 24.639402389526367, -34.377891540527344, 14.997819900512695, 4.023735046386719, 31.250391006469727, 55.64216613769531, -4.976271629333496, -2.331886053085327, -15.3844575881958, 15.465753555297852, 34.12117004394531, 19.146421432495117, 39.758522033691406, -9.797911643981934, -57.833003997802734, 47.73881912231445, 7.233229637145996, -14.584748268127441, -78.390380859375, 30.438758850097656, 12.563520431518555, 9.033855438232422, -8.648307800292969, -7.091741561889648, 14.184964179992676, 25.207168579101562, 40.91832733154297, 21.72834587097168, 37.97380828857422, 25.328542709350586, 30.565656661987305, 17.75568389892578, 21.395278930664062, 48.67436218261719, 34.3121223449707, -9.195576667785645, 0.1969524323940277, -6.565978527069092, 42.75733947753906, 13.317972183227539, -13.058398246765137, -28.696487426757812, -35.72297668457031, 11.513527870178223, -6.185255527496338, 4.8140106201171875, -46.892303466796875, -0.7396820783615112, 29.565738677978516, -32.804927825927734, -2.916034460067749, -5.432896614074707, -7.1470489501953125, -22.513917922973633, -21.511917114257812, 27.2240047454834, -66.01190948486328, 4.021113395690918, -14.914468765258789, -22.019033432006836, -8.483531951904297, -26.01673698425293, 45.68375778198242, 12.790070533752441, 29.442157745361328, -35.912750244140625, -41.695655822753906, -22.999923706054688, 45.059879302978516, 49.441524505615234, -2.7777953147888184, 9.833480834960938, -53.909061431884766, 7.486940383911133, -24.492855072021484, -16.343482971191406, -35.38656234741211, -6.106662750244141, -39.84086608886719, -3.275228261947632, 18.048357009887695, -40.112770080566406, 6.375953674316406, -2.3983495235443115, -18.688034057617188, 19.3188419342041, -11.255325317382812, -13.932816505432129, 22.70494270324707, 5.58195686340332, -52.93461608886719, 35.144081115722656, -61.449249267578125, -2.588759422302246, 44.93815612792969, 50.117156982421875, -22.038604736328125, -2.7038166522979736, 23.867427825927734, 21.4602108001709, 8.510125160217285, 48.2476806640625, -36.57884979248047, -22.770797729492188, -45.69596862792969, -29.05341339111328, 6.475618362426758, 57.21595001220703, 44.565223693847656, -7.912900447845459, 11.701468467712402, 40.415428161621094, -20.977306365966797, -8.740032196044922, 72.87989807128906, 43.4798698425293, 37.72929763793945, -32.909175872802734, 27.304258346557617, 48.531795501708984, 32.75377655029297, -50.97724533081055, 2.803276538848877, -49.169986724853516, 26.22441864013672, -31.068845748901367, -22.35013198852539, 0.6887314319610596, -15.521625518798828, 25.480728149414062, -26.2030029296875, -13.85329818725586, 10.844071388244629, -4.004027366638184, -16.98223304748535, -2.8673384189605713, 30.370426177978516, 14.004597663879395, 20.184141159057617, -51.54026794433594, 21.085439682006836, 36.66172409057617, 9.843963623046875, -51.6798095703125, -27.3548526763916, -28.150005340576172, -8.041732788085938, -7.234177112579346, -27.185588836669922, 5.979992866516113, 26.16412353515625, -61.60245895385742, 25.988862991333008, -2.0513696670532227, -43.27747344970703, -60.97190475463867, -31.085397720336914, -2.0402653217315674, -0.2194790244102478, -62.241512298583984, -99.41716003417969, 6.269396781921387, -27.953243255615234, -3.90564227104187, 7.845163345336914, 4.139585018157959, 5.940746307373047, -26.738204956054688, 30.889514923095703, 7.638235569000244, 11.876203536987305, 44.94644546508789, -35.075748443603516, -35.04531478881836, 40.95542907714844, -40.53450012207031, 28.009469985961914, -3.6568851470947266, -20.286035537719727, 24.84855842590332, 4.2700629234313965, -0.7890843749046326, -33.37773895263672, 25.465848922729492, 8.021273612976074, -13.513921737670898, 23.446693420410156, -18.721126556396484, -12.463678359985352, -44.31827163696289, -4.47987699508667, -0.35870280861854553, -4.066243648529053, -28.580585479736328, 9.26975154876709, 31.827072143554688, -72.17261505126953, -35.662261962890625, -3.1792209148406982, 6.157753944396973, -12.074053764343262, 49.735382080078125, 7.615206718444824, -9.100831985473633, -3.4815850257873535, 51.06660842895508, -16.117229461669922, 58.19236373901367, 8.096085548400879, 12.530853271484375, 21.404081344604492, -21.944889068603516, -35.79243469238281, 12.09214973449707, 25.96116065979004, -5.08858060836792, 21.143875122070312, 42.941768646240234, 9.839521408081055, -21.996530532836914, -36.669193267822266, 2.3270933628082275, -5.682427406311035, -69.29730224609375, -11.698325157165527, -28.039796829223633, -42.715938568115234, 28.44741439819336, -4.841939449310303, 41.12288284301758, -15.14595890045166, -36.606021881103516, -23.430898666381836, 15.576391220092773, 17.298315048217773, 11.394977569580078, -3.125589609146118, -34.239017486572266, -45.74383544921875, 18.21420669555664, -47.366939544677734, 44.4386100769043, -23.391664505004883, 33.149681091308594, -23.126792907714844, -52.24005126953125, 16.010744094848633, -16.82081413269043, -30.189523696899414, 22.66326332092285, -5.974989414215088, -35.31279373168945, -4.9723920822143555, -56.15607452392578, 4.226368427276611, -68.87348175048828, -15.885296821594238, -22.514631271362305, -10.123661994934082, -45.2476692199707, -52.375328063964844, 30.768184661865234, -66.2866439819336, -33.322235107421875, 2.806687355041504, -13.502046585083008, 10.693574905395508, -14.743824005126953, -60.40557098388672, -11.473773956298828, 54.501522064208984, -41.7083854675293, -14.199979782104492, -24.907617568969727, 37.92306137084961, -9.731861114501953, 23.85283088684082, -30.421274185180664, -38.34272003173828, -10.1321439743042, -20.926395416259766, 26.4917049407959, -12.432601928710938, 30.901498794555664, 0.012985676527023315, -34.84321594238281, 6.4240264892578125, -8.02824878692627, 10.91950798034668, -47.43875503540039, -33.36570358276367, -10.452874183654785, 30.415462493896484, 19.681468963623047, -3.2068824768066406, 8.258787155151367, -45.44406509399414, 6.002041339874268, 18.32880401611328, -10.557856559753418, 4.265970230102539, 4.99803352355957, 33.09186935424805, 33.08230972290039, -23.36878776550293, 12.376564979553223, 14.82226276397705, 31.530933380126953, -23.503162384033203, -0.7750409841537476, -68.81519317626953, 13.045560836791992, 35.60578155517578, 11.120145797729492, -23.4442081451416, -33.35367965698242, -12.736930847167969, 5.881913185119629, 25.079885482788086, 17.240673065185547, -63.856693267822266, 17.02682876586914, 47.093292236328125, -12.462114334106445, 22.208829879760742, 48.145057678222656, -13.705358505249023, 13.68271255493164, 13.968571662902832, -4.652020454406738, -41.30306625366211, -7.642767429351807, 2.255002498626709, -33.905452728271484, -52.39110565185547, -52.271202087402344, 10.201875686645508, 43.41923141479492, -9.859442710876465, -1.6003987789154053, 29.95478057861328, 1.8525452613830566, -19.371286392211914, -23.204792022705078, 41.645503997802734, -53.89265060424805, -27.699514389038086, 38.78971862792969, -33.4382209777832, 39.2202262878418, 53.373538970947266, 75.5909652709961, 29.2152042388916, 0.8525123000144958, 13.146780967712402, 52.14984130859375, -37.84328079223633, -9.971904754638672, -70.82518005371094, 26.710262298583984, 9.866327285766602, 14.447793006896973, -37.357566833496094, -15.574610710144043, -18.571081161499023, -28.353382110595703, -6.7991743087768555, 21.616233825683594, 50.94741439819336, -3.2465343475341797, -20.55011749267578, 34.66722106933594, -35.15907669067383, 38.67504119873047, -9.920854568481445, 89.7195816040039, 18.95461654663086, 12.460660934448242, -20.226869583129883, 3.251213788986206, -37.10568618774414, 36.950008392333984, 24.686059951782227, 28.831069946289062, 17.287094116210938, -63.55747604370117, 56.56950378417969, 20.639127731323242, -0.8204864263534546, -46.40129089355469, -18.809463500976562, -15.97523021697998, -37.82316970825195, -9.337745666503906, -11.637151718139648, 6.399940013885498, -28.871301651000977, 50.839813232421875, -19.967355728149414, -20.548715591430664, -77.89991760253906, 36.863372802734375, -9.862090110778809, 25.06343650817871, -39.496925354003906, 23.263267517089844, 26.780052185058594, -14.265073776245117, 1.0137076377868652, -55.355224609375, 12.414607048034668, -29.608774185180664, 4.814885139465332, -65.07765197753906, -40.78980255126953, -6.074859142303467, 22.830957412719727, 34.64815902709961, 22.871816635131836, 1.0900912284851074, 24.090957641601562, 4.211644172668457, 50.289737701416016, -56.20615768432617, 31.158506393432617, 45.01626205444336, 14.455357551574707, 25.537322998046875, -8.96672248840332, 15.46407413482666, -55.10658264160156, -21.575759887695312, -8.733803749084473, 3.453195810317993, -26.901439666748047, -1.7123229503631592, 16.749919891357422, 55.1699104309082, -31.733028411865234, -40.28715133666992, -33.75324249267578, 6.242276668548584, -11.710866928100586, 0.002233266830444336, -19.103618621826172, -12.715496063232422, 34.524139404296875, -4.1982421875, -28.20587730407715, 23.07501792907715, 41.10710144042969, -63.10102081298828, -12.159745216369629, -23.857492446899414, 21.895435333251953, -7.589578628540039, 8.333769798278809, 27.816089630126953, -51.923240661621094, 17.224559783935547, 46.81288528442383, 7.372992515563965, 27.135446548461914, 29.8269100189209, -27.170164108276367, 9.309356689453125, -3.530177593231201, -13.458330154418945, 6.784046173095703, 14.004836082458496, 38.65370178222656, 56.45035934448242, 20.393123626708984, -36.41741180419922, -5.674134254455566, -21.881546020507812, -21.90769386291504, 4.5957255363464355, 54.78885269165039, 6.890288352966309, 16.29940414428711, -51.95491409301758, -76.12545013427734, 13.466771125793457, -39.36103820800781, -34.820159912109375, -49.918479919433594, 25.68012809753418, 8.746150970458984, 6.404356956481934, 39.180328369140625, 1.8242387771606445, -14.346976280212402, 17.23329734802246, -11.931248664855957, 24.188081741333008, 35.03362274169922, -9.607565879821777, -38.936676025390625, 12.359952926635742, -30.223220825195312, 35.40813446044922, -5.877048492431641, -41.35240173339844, 34.555198669433594, 25.811355590820312, -15.627429008483887, -8.416488647460938, -11.1972074508667, -24.506132125854492, -25.86195182800293, 23.64509391784668, -23.677396774291992, 2.6558544635772705, 11.156288146972656, -31.144948959350586, 7.657196521759033, -3.4389538764953613, 49.48031234741211, 27.48714828491211, -15.038565635681152, 38.2196044921875, -35.50651550292969, 3.1040172576904297, 6.773726463317871, 62.42646408081055, -9.179056167602539, 14.955288887023926, -22.041927337646484, -14.043614387512207, -48.24605941772461, -1.6111513376235962, 4.457712650299072, -3.4221017360687256, 30.519025802612305, 0.5344536304473877, 22.3496036529541, 24.38841438293457, 28.154294967651367, -37.5529670715332, 0.06704843044281006, -51.14494323730469, -28.28461265563965, -3.959336280822754, 63.27631378173828, 18.408157348632812, -14.087501525878906, 46.654117584228516, 14.37723159790039, 26.625097274780273, -9.275947570800781, 35.4599723815918, -3.6157033443450928, -3.0340442657470703, 2.0622711181640625, 61.04023742675781, 10.496439933776855, -3.0711116790771484, 1.8607397079467773, 6.217370510101318, -31.858386993408203, -23.86396598815918, -21.73799705505371, 40.46830749511719, -15.776349067687988, -45.210140228271484, 33.21365737915039, -32.58046340942383, 22.122745513916016, 8.975658416748047, -25.70356559753418, 21.42246437072754, 21.09189796447754, -5.3196282386779785, 9.115549087524414, -28.937419891357422, 20.90140724182129, -10.70756721496582, 47.2502326965332, -16.481794357299805, 76.72554016113281, 20.430803298950195, -14.008955955505371, 13.390852928161621, -6.450013160705566, -8.541863441467285, -27.66231918334961, 23.843477249145508, 8.599871635437012, 37.250484466552734, 36.53269577026367, -6.761834621429443, 40.29448699951172, 18.789165496826172, 28.617849349975586, -14.93228530883789, 38.872520446777344, 3.769500732421875, 47.03229522705078, -19.554101943969727, -8.093610763549805, 7.019798755645752, -3.3150391578674316, -13.028017044067383, 11.464061737060547, -15.801826477050781, -34.98072052001953, -40.65458297729492, 13.546117782592773, 12.073016166687012, -22.62308692932129, -9.876736640930176, -2.2737460136413574, 10.69314956665039, 8.150474548339844, 53.05509567260742, 2.836359977722168, 3.8529343605041504, 38.26944351196289, -5.430891036987305, -18.10162353515625, 9.526213645935059, 106.43986511230469, -4.638546943664551, 7.556461811065674, -5.355942726135254, 4.506678104400635, 10.189813613891602, -41.71171188354492, 30.721723556518555, -32.11595916748047, 3.7451910972595215, -47.80079650878906, 33.476104736328125, -13.860438346862793, 45.77248764038086, -23.675264358520508, -10.085700988769531, -32.26905822753906, 19.505643844604492, 15.985461235046387, 18.87297248840332, -34.09113693237305, 8.331095695495605, -32.73427963256836, 7.384458065032959, -9.077218055725098, -20.39663314819336, -8.893162727355957, 24.23970603942871, -20.72188377380371, 13.574606895446777, 12.843546867370605, -18.938106536865234, -40.17066955566406, -5.869565010070801, -13.901630401611328, 40.45026397705078, 8.21833610534668, 20.041851043701172, -21.217199325561523, 42.10154342651367, 56.601295471191406, 10.437477111816406, 12.044900894165039, -8.385485649108887, 3.907705307006836, -20.84711265563965, -30.469654083251953, -42.25088882446289, -28.315349578857422, 42.6610221862793, -8.888811111450195, -39.13766860961914, 35.121177673339844, 71.6147232055664, -10.748125076293945, -29.278884887695312, 23.164600372314453, 43.09558868408203, -1.9689208269119263, -5.157519340515137, -43.84782028198242, -10.828024864196777, -22.14905548095703, 4.327600002288818, -42.128929138183594, -59.877140045166016, -18.98206329345703, 24.061870574951172, -20.011512756347656, 56.24455261230469, -21.67426109313965, 52.530677795410156, -32.10198974609375, 7.254075527191162, -37.57084655761719, -13.660613059997559, -11.075007438659668, -17.508216857910156, 35.28374099731445, -1.4779527187347412, 14.181771278381348, 35.400856018066406, 61.18263626098633, -46.96686553955078, 76.90558624267578, 23.61343765258789, 11.183699607849121, 25.552549362182617, -63.43257522583008, 26.632856369018555, 50.5207405090332, 44.36854553222656, -25.433584213256836, 36.68687057495117, -36.37554168701172, -21.722976684570312, -0.7491073608398438, -44.25791931152344, -65.72638702392578, 13.883983612060547, 0.403727650642395, -2.8951966762542725, 15.324173927307129, -7.437628746032715, 5.430749893188477, -32.09788131713867, -9.077920913696289, 51.887901306152344, 21.75797462463379, 40.916500091552734, -40.42033767700195, 3.1554455757141113, -5.177323818206787, 8.924983024597168, -55.43406677246094, -40.32850646972656, 8.26287841796875, 61.614559173583984, -10.654203414916992, 15.133456230163574, 1.5319663286209106, -9.139846801757812, 25.179075241088867, 9.377374649047852, 23.32080841064453, -0.3869398236274719, 22.78318214416504, -18.852252960205078, -3.5034093856811523, -32.60872268676758, -0.8594437837600708, -12.743779182434082, -3.7396488189697266, 42.4739990234375, 3.289957046508789, -10.980751037597656, 13.254061698913574, 24.111387252807617, 5.565845966339111, 38.07162857055664, -15.773475646972656, -7.997737407684326, -22.252334594726562, -1.9706448316574097, 33.5837287902832, -6.701006889343262, 76.4464111328125, 9.116270065307617, -32.284278869628906, 39.38718795776367, -0.36178159713745117, -14.819579124450684, -14.661391258239746, -65.83893585205078, -5.390838146209717, 28.110559463500977, 13.877202033996582, -31.264610290527344, -36.837459564208984, -16.741451263427734, 52.70954132080078, 16.036102294921875, 38.490272521972656, 16.16716957092285, 58.982208251953125, -25.833087921142578, 6.834758758544922, 2.1902949810028076, -22.530729293823242, 20.92185401916504, 22.28550148010254, -17.0993595123291, -4.085846900939941, 26.115230560302734, 77.0109634399414, 27.245426177978516, -1.5559865236282349, -2.0225369930267334, 2.3656094074249268, 4.8194899559021, -50.1024055480957, -5.98209285736084, -38.785911560058594, 83.37419891357422, 14.67486572265625, -6.321257591247559, -42.30934524536133, 0.7967545986175537, -28.796110153198242, 39.46067428588867, 12.718585968017578, 69.8436279296875, 14.473421096801758, 21.560688018798828, 38.18000793457031, 10.888398170471191, -11.22488784790039, 17.45206069946289, 41.02833557128906, 24.049137115478516, 14.801398277282715, 14.0329008102417, -6.1591796875, -10.717290878295898, -16.4449520111084, -19.355058670043945, 7.552797317504883, 35.79403305053711, 60.55494689941406, -17.148717880249023, 3.526198387145996, 9.878129005432129, 26.07693862915039, -14.77540397644043, -22.724552154541016, -35.04957580566406, 3.218759298324585, 27.2721004486084, 35.007041931152344, 3.108598232269287, 2.1634392738342285, -27.291168212890625, -48.26463317871094, -3.5373716354370117, 7.710264205932617, 3.3339550495147705, -14.25226879119873, -49.37519454956055, -14.674948692321777, -58.914180755615234, 28.206153869628906, 14.863151550292969, -14.621707916259766, 13.46386432647705, 36.03645324707031, 23.874326705932617, 0.29941990971565247, 43.5757942199707, 8.350024223327637, 7.6087493896484375, 15.878499984741211, -14.326810836791992, 38.68489456176758, -39.909584045410156, 13.853285789489746, -31.575681686401367, 7.5107245445251465, -49.413780212402344, 40.802772521972656, 17.630054473876953, 10.266355514526367, 32.70498275756836, -16.990596771240234, -36.273521423339844, -29.197093963623047, -2.8082125186920166, -1.6242330074310303, -23.49797248840332, -43.71870040893555, -6.931646347045898, 13.692000389099121, -5.71015739440918, 47.00894546508789, -27.62384033203125, 5.682160377502441, 37.049652099609375, -2.7639405727386475, -10.332489967346191, -31.27323341369629, -35.097808837890625, 68.93505096435547, 11.415878295898438, -59.582767486572266, -10.686321258544922, 8.495896339416504, 25.590822219848633, 61.84070587158203, 17.9633731842041, 8.143261909484863, 46.83742904663086, 50.33216857910156, 11.961414337158203, 27.65812110900879, 9.371268272399902, -41.86977005004883, -14.140774726867676, 40.363948822021484, 19.947500228881836, 55.74346160888672, -11.45116138458252, -21.05095672607422, 31.270156860351562, 63.93597412109375, -6.642972469329834, 69.95252227783203, 7.700380802154541, 40.518829345703125, 6.406856060028076, -0.7964489459991455, -37.36771774291992, 30.791746139526367, 27.45893096923828, -6.248510837554932, 23.88109016418457, 17.727170944213867, -34.325130462646484, -17.375699996948242, -2.430056095123291, 62.84497833251953, 10.491472244262695, -3.1578071117401123, 32.33152770996094, 34.65111541748047, -24.524112701416016, 13.759939193725586, -44.328609466552734, -41.102149963378906, 32.67034149169922, 18.157676696777344, 15.307110786437988, -16.029376983642578, -10.932463645935059, -48.5227165222168, -25.854801177978516, 1.6145148277282715, 27.358631134033203, 7.830682754516602, 63.760894775390625, 13.174947738647461, -21.57486915588379, 61.86830520629883, -23.43075942993164, -2.7290666103363037, 38.612098693847656, -27.126047134399414, -41.513404846191406, 1.5441217422485352, -40.46805953979492, -44.32033157348633, 46.77177429199219, 37.40361404418945, -7.5849480628967285, -13.864315032958984, -5.487855911254883, 10.28449535369873, 86.12105560302734, 19.953950881958008, -21.949670791625977, -23.658111572265625, 29.452730178833008, 56.46611785888672, 20.2299747467041, -48.26541519165039, -38.00658416748047, -5.5452985763549805, -13.50350284576416, -28.707366943359375, 62.933349609375, 15.279963493347168, 42.05689239501953, -48.05577087402344, 27.968944549560547, -4.483389854431152, 52.865867614746094, -12.25351333618164, 15.195921897888184, 12.376921653747559, -21.057228088378906, 25.35096549987793, -9.630392074584961, -4.083616256713867, -34.081111907958984, 17.629093170166016, -3.9028890132904053, -39.38828659057617, -30.044923782348633, 0.9170031547546387, -46.72121047973633, -16.450822830200195, 43.40389633178711, -29.165008544921875, -10.29723072052002, 74.85781860351562, 6.649154186248779, 13.251609802246094, 17.46986961364746, -0.11521577835083008, 17.78959846496582, 103.16625213623047, 30.019033432006836, 24.02956199645996, -11.620552062988281, -29.565317153930664, 14.184371948242188, -16.805992126464844, 40.619972229003906, -12.16501235961914, 13.398348808288574, -3.0161075592041016, -11.727254867553711, -9.715295791625977, 33.15342330932617, 22.259750366210938, -45.237728118896484, -13.894577026367188, -21.279808044433594, 22.953828811645508, -14.780685424804688, -17.72016143798828, -22.598630905151367, 42.58945846557617, 28.443286895751953, -59.318809509277344, -0.3648410439491272, -1.5070436000823975, 16.699438095092773, 14.492175102233887, 26.68655014038086, 66.32975006103516, -52.78935623168945, 50.209922790527344, -13.544931411743164, -42.65374755859375, -12.721366882324219, -29.401742935180664, 76.47600555419922, 21.344738006591797, 22.977296829223633, -9.113480567932129, 5.019893169403076, 26.815343856811523, 13.853740692138672, 17.792686462402344, 22.88799285888672, 25.046689987182617, -20.512563705444336, 36.41452407836914, -0.33767950534820557, -13.693401336669922, -29.01490592956543, 27.35486602783203, 14.30790901184082, 47.65608215332031, 4.206534385681152, -13.441144943237305, 10.96745777130127, -3.189018726348877, -17.185569763183594, -45.01863098144531, -31.654922485351562, 24.48638916015625, 13.9358549118042, -25.47043228149414, 11.962221145629883, 40.221527099609375, -5.482969284057617, 25.025571823120117, 11.167613983154297, 14.15115737915039, 21.667186737060547, -35.83742904663086, 21.15755271911621, 15.502204895019531, 59.32025146484375, -2.468743324279785, 40.94248580932617, 20.642101287841797, -20.619169235229492, -50.43111801147461, -25.679771423339844, -8.334134101867676, -9.82772159576416, 13.990568161010742, 18.41081428527832, -11.875664710998535, -17.902883529663086, -29.838790893554688, 80.5143814086914, 47.43593978881836, -42.285736083984375, 16.447416305541992, -9.580767631530762, 1.978708267211914, 60.32212829589844, -24.394184112548828, 4.357306003570557, -8.089452743530273, 32.734771728515625, 46.36064910888672, -9.281951904296875, -25.255191802978516, -7.393435955047607, -14.914458274841309, -60.165523529052734, -31.260894775390625, -31.473155975341797, -38.779296875, -36.72710037231445, -57.91227722167969, -13.852846145629883, -26.178712844848633, 3.694444179534912, -30.7466983795166, 25.191247940063477, -25.110797882080078, 26.304533004760742, -2.9082436561584473, 3.3826839923858643, -27.33843231201172, -22.758285522460938, 3.8654844760894775, -30.12196922302246, -17.617034912109375, 16.646923065185547, 6.648159980773926, -22.36372947692871, 29.251819610595703, -39.106170654296875, 20.168853759765625, 3.3988564014434814, 9.135289192199707, -13.26870346069336, 8.108181953430176, 21.921524047851562, -9.59352970123291, 21.480178833007812, -0.9376749396324158, 7.7494354248046875, 33.06108474731445, -16.563722610473633, 6.009243965148926, 11.854655265808105, 15.813809394836426, 46.39177703857422, 2.38063383102417, -3.3733887672424316, 78.58682250976562, -27.815631866455078, -55.4332389831543, -13.468082427978516, 23.136404037475586, -12.754825592041016, 58.24943542480469, 13.805051803588867, 5.969045162200928, 42.30624008178711, 9.188508987426758, -14.647148132324219, -2.7733521461486816, 12.54740047454834, 8.58968448638916, 15.113728523254395, -15.854451179504395, 23.4527530670166, -38.07028579711914, 6.449253559112549, 73.42330932617188, 34.4462776184082, 20.38789939880371, -24.99324607849121, -43.50509262084961, 60.684078216552734, 22.427894592285156, 19.537460327148438, 19.484106063842773, -8.516414642333984, 45.95368194580078, -40.35295104980469, -35.782413482666016, -5.70694637298584, 10.123807907104492, 17.93482208251953, -11.979626655578613, 27.45340919494629, -32.03402328491211, 6.358877658843994, 37.483699798583984, 2.5585474967956543, 34.77831268310547, 64.48603820800781, 30.435758590698242, 42.29270553588867, 66.93223571777344, 10.667206764221191, 12.060530662536621, 37.907684326171875, 12.196128845214844, -36.07740783691406, -64.25247955322266, 5.529135704040527, -29.900741577148438, -11.548178672790527, 16.034259796142578, -36.89638900756836, 17.446683883666992, 10.514996528625488, -48.941871643066406, -13.19322395324707, -15.050354957580566, 9.377795219421387, -16.25865364074707, -8.02099895477295, -15.493353843688965, -15.00700855255127, 51.622676849365234, -13.412440299987793, 45.671424865722656, 34.0418815612793, -31.612939834594727, -52.93640899658203, -38.38238525390625, -27.33914566040039, -4.8014655113220215, 50.16600799560547, 14.49213981628418, 20.25275993347168, 28.982158660888672, -0.05244000256061554, -1.2327336072921753, -5.2654242515563965, 27.588829040527344, -15.189352989196777, 10.922447204589844, -37.38050842285156, 54.04695129394531, -2.137270450592041, 0.5673424005508423, -57.489715576171875, 18.935148239135742, 14.94804573059082, 95.91218566894531, 21.880952835083008, -10.759932518005371, -30.803945541381836, -46.16747283935547, 44.15873336791992, 67.04654693603516, -22.539493560791016, 0.8716959953308105, -58.4310417175293, 7.904260158538818, 44.068763732910156, 64.12114715576172, -53.27078628540039, 7.46876335144043, -6.547045707702637, 21.821277618408203, 7.011105060577393, -24.440214157104492, -0.8904059529304504, 41.09019470214844, -94.7456283569336, -41.47804641723633, 39.49882125854492, -2.465622901916504, -60.04841995239258, 33.68582534790039, 12.550040245056152, 0.5777497291564941, 39.918087005615234, -30.503820419311523, 67.18244171142578, 42.16116714477539, 70.7939453125, 5.053617477416992, -30.017189025878906, -12.417624473571777, -45.21817398071289, 26.836170196533203, 28.061767578125, -0.8522692322731018, 2.5787010192871094, -4.2077436447143555, 1.8130232095718384, -12.829191207885742, 58.7695426940918, -68.63966369628906, -30.512256622314453, 42.027099609375, 35.938720703125, -21.294513702392578, 1.8602025508880615, -7.801088333129883, 7.002217769622803, -22.877643585205078, -36.059139251708984, -5.4964799880981445, 21.38546371459961, 10.22675895690918, -31.922380447387695, 19.191970825195312, -16.64076805114746, -10.309745788574219, -68.1787338256836, -13.273270606994629, 29.821779251098633, -29.077342987060547, -20.269006729125977, 9.252776145935059, 36.410865783691406, 36.72987365722656, -3.5339412689208984, -29.914953231811523, -25.820486068725586, -17.77089500427246, -8.397309303283691, -3.5810041427612305, 61.401432037353516, 27.045190811157227, -63.0079460144043, 27.178112030029297, 2.4428415298461914, 30.786706924438477, 38.13020324707031, -18.081466674804688, 43.36011505126953, -1.2043994665145874, 54.475852966308594, 12.814262390136719, 5.430695533752441, -20.648902893066406, 3.3443737030029297, 10.701258659362793, 15.565021514892578, -3.9610610008239746, -32.99386215209961, -28.178401947021484, -10.07492446899414, 25.008865356445312, 22.37163543701172, -25.019304275512695, -40.42120361328125, -24.324729919433594, 16.740867614746094, -2.85922908782959, 12.96291732788086, 8.070645332336426, 6.944556713104248, 50.98014831542969, 8.181374549865723, -40.89498519897461, 27.301837921142578, -46.51589584350586, -14.607762336730957, -61.130069732666016, -1.2452778816223145, -60.60337448120117, -41.148277282714844, -21.69434928894043, 3.0343542098999023, 20.284442901611328, -61.84545135498047, 26.916034698486328, -15.762127876281738, -8.79586124420166, -3.9496288299560547, -39.457767486572266, 39.536102294921875, -74.74411010742188, -45.748497009277344, 6.263976097106934, 16.21988296508789, 8.290251731872559, 0.36439061164855957, -23.907058715820312, -0.2615762948989868, -12.095051765441895, -54.546775817871094, 30.962223052978516, -4.869010925292969, 33.21334457397461, 13.939176559448242, -25.571853637695312, -45.13883590698242, 23.105854034423828, 18.52693748474121, -19.08047866821289, -19.395959854125977, -28.692962646484375, -14.711648941040039, 52.69428634643555, -9.472386360168457, 39.578887939453125, 21.823087692260742, -29.288806915283203, -12.7841215133667, -59.872108459472656, -18.184364318847656, 39.915950775146484, 26.148637771606445, 8.010886192321777, -18.83900260925293, 23.60582160949707, -45.03174591064453, 3.184934377670288, 14.6149263381958, 22.908042907714844, 3.4288413524627686, -11.639305114746094, 39.813297271728516, -17.881519317626953, 22.631179809570312, 5.249212741851807, 5.325669765472412, 62.87705993652344, -3.2992799282073975, 2.54412841796875, 27.275287628173828, 23.053409576416016, -29.284334182739258, 52.24978256225586, 11.960748672485352, 5.324889183044434, -6.985195636749268, 29.832454681396484, -60.112274169921875, -11.022391319274902, 9.247528076171875, 5.632392883300781, -82.31736755371094, 58.22798156738281, 20.643661499023438, -9.137533187866211, 11.322571754455566, -17.25503921508789, 18.513334274291992, -26.36663818359375, -16.736976623535156, -18.509130477905273, -16.829221725463867, -11.994221687316895, 16.448440551757812, 26.48650550842285, 43.477508544921875, -1.748127818107605, 10.345428466796875, 23.324108123779297, -40.6220817565918, -29.647117614746094, 11.369065284729004, 15.68201732635498, -20.20795440673828, -51.07478713989258, 4.502124309539795, -33.612430572509766, 12.100897789001465, -25.62471580505371, 38.242549896240234, -88.64762115478516, -30.56536102294922, 26.182514190673828, -3.453526020050049, 8.019742965698242, 33.53123474121094, -16.670454025268555, 37.47917556762695, 11.686819076538086, -18.344982147216797, -23.597272872924805, -3.9448277950286865, 17.389638900756836, -56.77070236206055, 16.13123321533203, 12.756753921508789, 44.62420654296875, -30.053258895874023, 20.487178802490234, 38.657859802246094, -0.5580077171325684, 48.77967834472656, -2.2625200748443604, 7.007538795471191, -33.16670227050781, 23.363086700439453, -32.9327392578125, 48.329715728759766, 7.809047698974609, 15.673011779785156, 3.088160514831543, -37.394432067871094, 4.376166343688965, 51.095428466796875, -22.851781845092773, 14.362196922302246, -54.09191131591797, 11.857434272766113, 15.066144943237305, 0.03251868486404419, -30.61270523071289, -32.03873825073242, -32.16546630859375, 12.429046630859375, 20.67064094543457, -33.14533233642578, 16.96944808959961, 22.664175033569336, 5.03038215637207, -24.254201889038086, -3.122692108154297, 0.36575353145599365, -34.518592834472656, -23.2537784576416, 10.593524932861328, 9.131549835205078, -12.215958595275879, 11.909942626953125, 21.55605697631836, -45.270164489746094, 55.766117095947266, -30.214059829711914, -31.583084106445312, -23.818592071533203, 8.888948440551758, -4.302706241607666, 3.89449405670166, -9.61307144165039, -24.41935157775879, 75.44214630126953, -32.058799743652344, -14.98756217956543, -22.050962448120117, -28.387006759643555, 50.54179000854492, -13.159483909606934, 28.92408561706543, -21.873685836791992, -29.63798713684082, -33.0605583190918, -29.468690872192383, 3.0505716800689697, -18.375350952148438, 2.6484384536743164, -45.741363525390625, 59.59971618652344, -6.2176642417907715, 7.392046928405762, 37.16742706298828, -10.289046287536621, 19.599550247192383, 12.422518730163574, -10.167469024658203, 30.367666244506836, -60.20343017578125, -70.10557556152344, -28.809186935424805, -49.644596099853516, 38.047950744628906, -30.490175247192383, 9.013866424560547, -42.537166595458984, -50.97534942626953, -27.274993896484375, -39.20505905151367, -40.92227554321289, 20.636857986450195, -37.43062973022461, 50.78040313720703, 0.5870818495750427, 28.49373435974121, 50.046627044677734, 12.09587287902832, -3.2541327476501465, -25.33686065673828, 12.958885192871094, 39.3700065612793, -38.109710693359375, -72.91897583007812, -26.666698455810547, -39.927711486816406, 0.515704333782196, -6.118659973144531, 1.972405195236206, 30.39580726623535, -18.403114318847656, -41.66769790649414, 9.244882583618164, 43.15711975097656, 17.028858184814453, 37.515193939208984, 11.735664367675781, -8.163243293762207, 5.296658992767334, -46.79229736328125, 10.64756965637207, 6.609176158905029, -32.491363525390625, 21.518516540527344, -20.192646026611328, 7.4899725914001465, -2.505718231201172, -1.3692026138305664, 23.77808952331543, -25.59951400756836, -12.610387802124023, -39.467803955078125, -32.446006774902344, 11.5135498046875, -19.928550720214844, 34.00528335571289, -25.630218505859375, -13.182223320007324, -38.08358383178711, 36.291969299316406, 30.802120208740234, -9.23438549041748, -33.50980758666992, -10.996699333190918, -54.140567779541016, 21.254722595214844, -13.991697311401367, -78.57756805419922, 20.228147506713867, 3.8769078254699707, -15.83827018737793, -6.534139633178711, -5.827192306518555, 42.017581939697266, 5.284574031829834, -34.665855407714844, -29.661882400512695, 60.032989501953125, -6.052069187164307, 10.651561737060547, -32.39677429199219, 24.15841293334961, 30.574203491210938, -25.10073471069336, 20.909358978271484, 37.57401657104492, 24.469942092895508, -23.765748977661133, -19.320894241333008, -20.504776000976562, -50.08861541748047, 27.93535041809082, -6.238197326660156, -16.713804244995117, -63.82188415527344, 18.223594665527344, -13.021081924438477, 5.960165023803711, -17.948993682861328, -34.90192413330078, 10.604961395263672, -41.1452522277832, -14.324077606201172, 4.777017593383789, -38.581546783447266, 30.567584991455078, -2.4265480041503906, -3.8160152435302734, 5.161786079406738, -8.761290550231934, 9.770299911499023, 12.241744041442871, 35.89122772216797, 15.182819366455078, -39.927371978759766, -14.735982894897461, -22.983362197875977, -31.44609832763672, 26.67662239074707, 54.865882873535156, 6.256278991699219, -18.613779067993164, -17.742740631103516, -5.039154052734375, 2.185467004776001, -23.173126220703125, -3.353921890258789, -12.546717643737793, 32.679901123046875, -15.824029922485352, 31.775249481201172, 16.15873908996582, -7.593250751495361, 14.481588363647461, 10.259818077087402, -68.78382110595703, -29.681493759155273, 6.34671688079834, -42.97434616088867, 27.775177001953125, -1.5011341571807861, 24.012596130371094, -6.910942554473877, -7.842625617980957, -18.455724716186523, -28.018150329589844, -13.770589828491211, 51.29549789428711, 44.12065887451172, -15.86115550994873, 4.989526271820068, 39.3990364074707, -14.962494850158691, 34.562095642089844, 35.19310760498047, -20.4229736328125, -19.987443923950195, -2.360196352005005, -46.65868377685547, -41.46385192871094, -9.389494895935059, -34.97835159301758, 0.12083315849304199, 31.27016830444336, 32.70425033569336, 41.059383392333984, 13.065657615661621, 11.673510551452637, 27.287593841552734, -48.79976272583008, 5.473309516906738, 22.291929244995117, -11.812647819519043, 2.969848871231079, 69.69471740722656, -11.948081970214844, 22.619239807128906, -4.439122200012207, 78.87117004394531, -19.718090057373047, -35.82120895385742, -21.8810977935791, -18.195507049560547, 13.032236099243164, 16.225242614746094, 7.341702938079834, -38.51906967163086, 12.92101764678955, 3.4870800971984863, -29.51902198791504, -13.611225128173828, -8.479667663574219, -29.875917434692383, -16.85820198059082, 0.7421183586120605, 16.6553897857666, -2.4670188426971436, -24.928096771240234, -36.33417892456055, 31.593603134155273, 71.92239379882812, -28.91280746459961, -42.581180572509766, -26.971628189086914, -64.04689025878906, -34.94292449951172, -11.060437202453613, -39.6787223815918, 24.872276306152344, 1.0975308418273926, -38.08229064941406, 5.564212799072266, 31.029464721679688, 1.0746545791625977, -10.398580551147461, 21.935794830322266, -4.706736087799072, -37.146270751953125, 8.150339126586914, 13.13571548461914, 27.872928619384766, -30.037982940673828, -46.81060028076172, 18.544187545776367, 19.708036422729492, -21.53948402404785, -8.945752143859863, -37.26150894165039, 5.981581687927246, 15.511148452758789, 100.12623596191406, -3.5841705799102783, -1.6708440780639648, 10.666391372680664, -43.973453521728516, -33.68919372558594, 33.541194915771484, 26.434724807739258, 3.2000513076782227, -16.43342399597168, 10.301024436950684, 56.81734848022461, -19.102787017822266, 41.1222038269043, -11.625346183776855, -46.15385437011719, 3.8957743644714355, 13.242392539978027, 22.669076919555664, -26.749380111694336, 10.701364517211914, -15.76518726348877, -21.992595672607422, 20.83951187133789, 51.7459716796875, -32.31189727783203, 19.571731567382812, 4.409313201904297, -9.574494361877441, -25.432886123657227, 19.788511276245117, -73.80154418945312, 29.374975204467773, -2.8501501083374023, 3.8345673084259033, 11.02505874633789, -48.716461181640625, -18.22602081298828, -33.2608528137207, -5.281486988067627, -76.47702026367188, 15.170937538146973, 40.17778015136719, -1.1120073795318604, 24.895841598510742, 28.43674087524414, -12.869829177856445, 33.05952072143555, 28.297069549560547, -12.795533180236816, -25.444965362548828, 9.015775680541992, -22.179777145385742, 12.935053825378418, -46.364131927490234, -48.34987258911133, -11.560619354248047, 41.87784194946289, -1.1677683591842651, -17.68581771850586, 72.00141143798828, 106.06254577636719, 31.99629783630371, 21.850046157836914, -10.51593017578125, 40.105857849121094, -43.32122039794922, -31.928735733032227, -16.555252075195312, 38.52267837524414, -15.51611328125, -17.500530242919922, 22.65644645690918, -15.10766315460205, -26.340782165527344, -18.999038696289062, -23.777467727661133, 14.908242225646973, -12.583369255065918, 31.480255126953125, 10.58106803894043, -51.43449783325195, -28.829343795776367, -17.03478240966797, 3.567244529724121, -17.349231719970703, 10.9557466506958, -13.533164024353027, 1.9678908586502075, 0.32957613468170166, 15.61060905456543, 9.937652587890625, -14.753408432006836, 19.912521362304688, 31.223045349121094, -17.86575698852539, 40.895362854003906, 66.85874938964844, -9.204922676086426, -25.507102966308594, -8.140580177307129, 19.73960304260254, 28.22504234313965, 56.86991500854492, 18.659587860107422, -4.875455856323242, 2.4907665252685547, 32.76295471191406, 2.2458577156066895, -3.095269203186035, 34.11662292480469, -46.72795486450195, -22.307165145874023, -12.323263168334961, 20.62530517578125, 6.5343146324157715, -26.951204299926758, 73.07946014404297, 33.85109329223633, 46.260501861572266, -15.1948881149292, 6.019232749938965, 18.60687255859375, -46.89173889160156, -20.46599578857422, -17.343658447265625, 8.403353691101074, -51.412376403808594, -9.616653442382812, 21.823875427246094, 10.286834716796875, 1.1105397939682007, -7.568498611450195, -0.17244622111320496, -38.06916427612305, 43.96031188964844, -9.6114501953125, -10.516508102416992, 31.592870712280273, 0.9514778852462769, -25.75895881652832, -12.497709274291992, 50.03172302246094, 59.01985549926758, 27.491397857666016, 13.556395530700684, 12.558876037597656, -21.074726104736328, 37.99278259277344, -47.93612289428711, -55.759700775146484, 4.309931755065918, -58.178550720214844, 37.369606018066406, -48.026126861572266, -17.383481979370117, 17.68003273010254, -6.234790325164795, -8.253436088562012, 38.72927474975586, -48.675472259521484, 5.893363952636719, 29.91330337524414, 12.87241268157959, -25.639616012573242, -0.04636836051940918, 10.863387107849121, 4.960138320922852, -2.7881689071655273, 36.30727767944336, 50.57698059082031, 2.4747471809387207, -13.16788387298584, -47.42385482788086, -43.8407096862793, 17.723798751831055, -43.86738204956055, 55.67350769042969, -3.2652587890625, 18.44049072265625, 39.710018157958984, 13.12769889831543, -52.42343521118164, -38.0803108215332, -4.440624237060547, -27.36659049987793, 39.55286407470703, 35.07640075683594, 8.186180114746094, 14.565999984741211, 30.068857192993164, -13.72416877746582, 13.153712272644043, 6.594989776611328, 48.735931396484375, -22.707136154174805, -43.72658157348633, -0.863612174987793, 65.59003448486328, 27.852359771728516, 25.77478790283203, 25.944049835205078, -32.0889892578125, -37.8437385559082, 25.644927978515625, -12.262834548950195, 28.635408401489258, -65.99793243408203, 7.04527473449707, -27.366897583007812, -6.142551422119141, -17.556617736816406, 32.95670700073242, 16.541332244873047, -20.6796875, 0.7396372556686401, -25.410429000854492, 57.309715270996094, 56.63245391845703, 29.51422119140625, -18.733259201049805, 15.722092628479004, -39.02885437011719, -29.06441879272461, -29.909753799438477, -4.064810752868652, -51.744056701660156, -37.43556213378906, 8.511208534240723, 3.869096279144287, -6.098127365112305, 16.305885314941406, -37.895347595214844, -26.018049240112305, -9.32436752319336, -11.5042142868042, 6.151050567626953, -28.160886764526367, -28.17009925842285, -33.33246994018555, 39.13919448852539, 56.507179260253906, -0.7730697989463806, -11.403924942016602, -7.6219401359558105, 6.883284091949463, 28.75846290588379, 7.656454086303711, -24.53157615661621, 68.75556945800781, -36.08582305908203, -7.021481037139893, 21.847036361694336, 42.688533782958984, -4.572099685668945, -14.507469177246094, -46.10569763183594, 10.595495223999023, 16.209749221801758, 36.64726257324219, 18.005247116088867, -6.8994855880737305, -6.337786674499512, 48.524898529052734, 30.14281463623047, -16.056716918945312, -17.279817581176758, -9.311310768127441, 13.268258094787598, 32.560977935791016, 1.0808441638946533, 26.76068878173828, 1.155092477798462, 6.825278282165527, 56.4844970703125, -32.040714263916016, -13.42998218536377, 19.705121994018555, -40.660335540771484, -80.6611557006836, -11.976451873779297, 11.189807891845703, -28.91498565673828, -12.709732055664062, 7.3608198165893555, 0.009168818593025208, 35.44857406616211, 26.271041870117188, 1.318676233291626, 37.96771240234375, -8.047959327697754, -0.09005509316921234, -6.758411884307861, 6.311404705047607, 12.762262344360352, -60.53851318359375, 31.119403839111328, -45.21323013305664, -11.81181812286377, -21.155731201171875, -37.54983139038086, -36.787811279296875, 6.590986728668213, -35.22652816772461, 55.14920425415039, 36.506805419921875, 1.9299687147140503, -32.152130126953125, -2.1127688884735107, 12.391127586364746, 7.422787189483643, 45.337833404541016, 52.38835906982422, -4.568377494812012, -13.946025848388672, 11.055103302001953, -7.9197096824646, -22.945985794067383, -32.897911071777344, 21.905561447143555, 14.642782211303711, -24.899242401123047, 53.375343322753906, -41.425880432128906, -40.33426284790039, -0.7513909339904785, 24.49264907836914, 22.27482795715332, 33.32221984863281, 6.762009620666504, -25.141387939453125, -2.1810450553894043, 4.71380615234375, 3.2682409286499023, -28.5577449798584, 17.836395263671875, -43.17230224609375, 25.903121948242188, 42.25286102294922, 43.636451721191406, -61.263587951660156, 41.35774230957031, 12.516755104064941, 27.348237991333008, 29.258291244506836, -28.405656814575195, -43.028343200683594, -24.534732818603516, -10.970725059509277, 52.971561431884766, 0.9205228090286255, -5.258934497833252, 30.63992691040039, 12.76441764831543, 51.666996002197266, 31.888408660888672, 60.1914176940918, 33.892845153808594, -24.644588470458984, 39.27018737792969, -22.577177047729492, 21.11479949951172, -4.730948448181152, 18.36144256591797, 5.418917655944824, -14.971115112304688, 31.300533294677734, -21.40894317626953, 13.223430633544922, 4.772197723388672, 3.159008502960205, 31.69991111755371, 0.005102992057800293, -12.09259033203125, 0.41145059466362, 38.097469329833984, -29.321638107299805, 17.311527252197266, -15.656168937683105, 51.17851638793945, 7.8672614097595215, -11.382067680358887, -17.906055450439453, 38.9444580078125, -24.569150924682617, -0.6766372919082642, -34.29331588745117, 26.324281692504883, 22.82520294189453, -23.699012756347656, -32.550418853759766, 1.5802475214004517, -22.155458450317383, -0.9331120848655701, -57.23073196411133, -16.861682891845703, 31.48258399963379, -13.002470970153809, 10.842589378356934, 13.248398780822754, -53.688846588134766, 3.194465160369873, 19.88665008544922, 25.421875, -33.46345520019531, 27.578027725219727, 26.27562141418457, -28.5063419342041, -29.447988510131836, 6.2596659660339355, 41.74894714355469, 45.142059326171875, -50.45896530151367, -51.713741302490234, 60.283348083496094, -22.520111083984375, 42.73154830932617, -37.478694915771484, 17.247745513916016, -1.5667166709899902, -22.180274963378906, 3.225769281387329, -5.424191951751709, -39.78356170654297, -13.281231880187988, -34.77214050292969, -9.811635971069336, 3.4510719776153564, 1.3487402200698853, -26.46900749206543, -31.02388572692871, 23.25830841064453, -12.776439666748047, 19.143367767333984, -6.1092987060546875, -17.158485412597656, 34.28012466430664, -22.743408203125, -52.674583435058594, 4.647305488586426, 0.5694700479507446, -15.419941902160645, -3.8292272090911865, 9.366034507751465, -38.061302185058594, -1.6620488166809082, -20.060670852661133, -9.780905723571777, 17.57645606994629, 24.438138961791992, -14.237295150756836, -16.24853515625, -11.93239974975586, -21.35784339904785, 19.25752830505371, -16.769067764282227, 13.847099304199219, 21.723735809326172, 50.44290542602539, 5.911902904510498, -43.80756378173828, 4.000441074371338, -8.730354309082031, -10.547758102416992, 15.001155853271484, 3.6797242164611816, 1.363644003868103, 1.3262555599212646, 5.128649711608887, -51.07184600830078, 16.046249389648438, -63.19316101074219, -2.491177558898926, 27.64468002319336, -45.52811813354492, 7.064311981201172, 21.086483001708984, -4.34779691696167, 2.2861974239349365, -57.469966888427734, -15.771599769592285, 26.35527992248535, -23.295557022094727, -17.971378326416016, 28.435405731201172, -2.045219659805298, 31.43696403503418, -17.46217918395996, -29.02320098876953, -9.966865539550781, 29.39327621459961, 29.786287307739258, 10.496467590332031, 7.014573097229004, -1.3249058723449707, -12.706418991088867, -8.092395782470703, -19.98274040222168, 2.3626515865325928, -11.714447021484375, 29.103029251098633, 52.73663330078125, 36.450714111328125, -22.059123992919922, 5.247986793518066, 18.680513381958008, 6.156342506408691, 1.1871765851974487, -9.03860855102539, 9.744010925292969, -38.762107849121094, 53.66059494018555, -11.45150089263916, 6.923208713531494, -27.160179138183594, 0.7633232474327087, -36.62559509277344, -26.13918113708496, 11.564526557922363, 42.10751724243164, -60.354305267333984, 16.52803611755371, 28.931049346923828, -3.8678338527679443, -9.053688049316406, 11.305394172668457, -67.2408447265625, -36.14224624633789, -17.94858741760254, -17.734359741210938, -9.154180526733398, 58.11967086791992, 10.918618202209473, 7.886210918426514, 6.895259380340576, -22.225011825561523, -21.733789443969727, 7.131716728210449, -19.29377555847168, -6.632568359375, 18.513208389282227, 30.92984962463379, 18.374454498291016, -1.8033509254455566, -54.2314453125, -2.461851119995117, -19.063684463500977, 42.63071060180664, -29.698015213012695, -12.072288513183594, -4.010662078857422, -6.784371376037598, 19.298282623291016, -6.988390922546387, -41.4854850769043, 11.1493558883667, -33.830413818359375, 63.77449417114258, 5.327464580535889, 27.942319869995117, 27.077606201171875, -13.255892753601074, -42.327003479003906, -17.36630630493164, -27.62202262878418, 57.74626922607422, 14.808316230773926, -22.642541885375977, 49.363712310791016, -10.220589637756348, 36.392845153808594, 44.75083541870117, -14.43167781829834, -17.27484703063965, 13.080034255981445, -0.09130096435546875, -63.13228988647461, -10.753403663635254, 6.175601482391357, 4.060128688812256, -38.91856384277344, -12.829254150390625, -23.220903396606445, -13.071094512939453, 44.893157958984375, -26.145044326782227, 27.906944274902344, -6.600757122039795, -55.04297637939453, 34.86402130126953, 15.126493453979492, -31.317602157592773, 87.26702117919922, -13.720139503479004, 5.881063461303711, -30.109039306640625, -45.961971282958984, -52.1671257019043, 23.968456268310547, -36.011268615722656, -85.21784210205078, -2.2302112579345703, -43.6752815246582, 63.15262985229492, -1.9358807802200317, -38.63479995727539, 9.416887283325195, 15.747632026672363, 7.651735782623291, 2.7624363899230957, -7.1808319091796875, -25.528656005859375, 47.0667839050293, 2.515986680984497, 20.167497634887695, 29.758251190185547, 7.637668132781982, 44.60280990600586, 30.602933883666992, 11.372811317443848, -31.304658889770508, 0.03135189414024353, -27.482467651367188, 7.139340877532959, -1.3375484943389893, 22.50035858154297, -6.788597106933594, -22.976451873779297, 2.5587832927703857, 7.01745080947876, -14.408748626708984, -2.871715784072876, 2.759784698486328, 20.155961990356445, 5.801970481872559, 12.943684577941895, -19.506532669067383, 48.05527877807617, 9.836709976196289, -43.76299285888672, -30.905670166015625, -6.455594062805176, 62.259490966796875, -69.335205078125, 29.052257537841797, 14.065634727478027, 34.46479797363281, -34.97525405883789, -34.917964935302734, -26.844892501831055, 21.511693954467773, -52.052398681640625, 10.446736335754395, 31.06563949584961, 26.305299758911133, -26.613046646118164, -19.0120849609375, -23.854530334472656, -24.749879837036133, -6.607583999633789, -1.8979943990707397, 24.848037719726562, 34.865257263183594, 11.734350204467773, 54.152488708496094, 55.56565856933594, 25.780073165893555, -0.16347283124923706, 30.013160705566406, 13.90241813659668, -27.549402236938477, -5.548611640930176, 8.255899429321289, 15.20321273803711, -46.29319381713867, 26.73345184326172, -55.4853630065918, 31.262081146240234, -30.576923370361328, 35.333438873291016, 29.346532821655273, -58.44693374633789, -27.22857666015625, -27.481063842773438, -49.15068817138672, 41.7747688293457, 10.023397445678711, 70.2155532836914, 8.841240882873535, -25.402372360229492, 46.05000305175781, 33.50117492675781, -26.521106719970703, 11.593870162963867, -25.390342712402344, 29.658687591552734, 40.576541900634766, -4.796854019165039, -31.555034637451172, 9.330677032470703, 5.450460910797119, 4.7085981369018555, -18.42576026916504, -0.9393237233161926, 17.674409866333008, -28.315540313720703, 13.587966918945312, 36.3106689453125, 15.187721252441406, -37.14370346069336, -2.0612716674804688, 7.160539627075195, 17.6429443359375, 44.07748031616211, 40.38661575317383, -1.0506885051727295, -36.38627243041992, -17.326557159423828, -3.7224674224853516, -1.0759105682373047, 3.3315281867980957, -18.32305908203125, -30.589113235473633, 43.72785568237305, 36.301815032958984, -3.1510086059570312, 29.254112243652344, -39.909698486328125, -49.14519119262695, -3.6240196228027344, -56.868064880371094, -11.961028099060059, 50.46981430053711, -21.784381866455078, 24.592775344848633, 69.04705810546875, 9.718655586242676, 2.4131128787994385, -27.732383728027344, 18.613773345947266, -21.321481704711914, 43.709388732910156, 36.8712043762207, -30.174402236938477, 18.319101333618164, -34.11882781982422, -7.416992664337158, -48.229644775390625, -2.8183579444885254, -6.843635559082031, -6.276191234588623, -37.798912048339844, -26.721647262573242, 50.64374542236328, 52.87430953979492, -53.486000061035156, -16.321311950683594, 14.76120662689209, -16.661273956298828, 33.7921257019043, -6.991412162780762, 25.368242263793945, -12.832903861999512, 34.757991790771484, 34.563751220703125, 26.325851440429688, 24.618661880493164, 46.04489517211914, -12.160051345825195, 33.14163589477539, -6.055511474609375, 24.394906997680664, 8.182385444641113, 15.756070137023926, 20.249330520629883, 33.42259216308594, 35.133644104003906, -40.21600341796875, 29.80754280090332, 8.876321792602539, 3.139209270477295, 29.404924392700195, 18.501741409301758, 24.346466064453125, -6.43721342086792, 73.4583511352539, -4.338168621063232, -23.500324249267578, -10.55147933959961, -17.39160919189453, -41.08108139038086, 6.3601579666137695, 29.919551849365234, -15.304544448852539, 20.664325714111328, -0.5800822973251343, 58.46587371826172, -48.10249710083008, -63.31344223022461, -2.870358943939209, -12.18159294128418, -46.99977493286133, -12.633210182189941, -15.842094421386719, 2.7387049198150635, 23.210193634033203, -34.80477523803711, 9.22232437133789, -51.73515701293945, -65.97333526611328, -32.704322814941406, -39.093772888183594, 17.967342376708984, 32.468727111816406, 14.858208656311035, 56.370948791503906, -6.6044087409973145, -5.025690078735352, -38.580848693847656, -13.660881996154785, -5.5177412033081055, 21.070985794067383, 8.939598083496094, -44.733787536621094, -39.5584602355957, -6.339655876159668, 49.00061798095703, 41.47952651977539, -1.327049732208252, -40.06123352050781, 5.761809825897217, -57.145774841308594, 62.63278579711914, 4.631036758422852, -25.584653854370117, -12.17786693572998, -8.528350830078125, -12.191822052001953, -33.02585220336914, -74.83491516113281, 19.915752410888672, -49.5592041015625, 22.278396606445312, -103.08638763427734, -24.778324127197266, 21.56719398498535, 35.20689010620117, 24.353891372680664, -48.61170959472656, 8.74422836303711, -19.15172004699707, 7.481241703033447, -20.76273536682129, 26.908790588378906, -8.356161117553711, 38.18225860595703, -11.737136840820312, -40.112178802490234, 45.037776947021484, -32.28308868408203, -5.165401458740234, 8.719437599182129, 50.52152633666992, 15.367936134338379, 42.10601043701172, 10.910364151000977, 73.00263214111328, -4.157577991485596, -3.9198501110076904, -11.773350715637207, 3.8647615909576416, -18.599218368530273, 15.227581977844238, -30.57775115966797, -16.1761531829834, 7.03482723236084, 7.248735427856445, -51.98373031616211, -15.605185508728027, -44.30461120605469, 44.470096588134766, 12.057300567626953, 7.518821716308594, 5.483623027801514, -46.477455139160156, 3.0702567100524902, 44.57050704956055, 3.906519889831543, -1.440268635749817, 11.212852478027344, 3.3112428188323975, 37.24388885498047, -18.089841842651367, 23.048254013061523, 47.56027603149414, -19.636812210083008, -19.516010284423828, -20.094741821289062, 15.967586517333984, 8.809395790100098, 21.33180809020996, -10.393074989318848, 21.91657829284668, -2.5370841026306152, -25.344234466552734, -38.906375885009766, 53.16645431518555, -17.131710052490234, -1.856201410293579, -35.137290954589844, -8.00840950012207, -2.976168155670166, 25.63576889038086, -24.223581314086914, 7.932735919952393, 11.897164344787598, 21.17507553100586, 26.736478805541992, 73.130126953125, -22.729389190673828, 11.780985832214355, -43.11939239501953, 49.74899673461914, -26.588897705078125, 27.55010414123535, 9.33752727508545, -9.470580101013184, 44.04351806640625, -0.2171485424041748, 42.286922454833984, 23.895681381225586, -30.797700881958008, 2.6981778144836426, -19.378129959106445, 50.12732696533203, 18.232545852661133, 16.163652420043945, -27.25412940979004, 31.271812438964844, -49.725860595703125, 1.1341581344604492, 0.722701907157898, -24.3541259765625, 13.932467460632324, 4.679335594177246, -0.6035860180854797, 3.6979455947875977, 43.564910888671875, -15.708229064941406, 58.34678649902344, 47.38312911987305, -76.20140838623047, -11.037088394165039, 2.149589776992798, 31.64448356628418, -15.070420265197754, 10.648294448852539, -9.238995552062988, 42.1163444519043, 2.2953832149505615, -20.999797821044922, 68.68049621582031, -3.3268589973449707, -31.68622398376465, 35.95716857910156, -24.340124130249023, -7.694487571716309, 17.55236053466797, -3.1762773990631104, -11.518338203430176, -12.006575584411621, 4.647647857666016, -6.9009857177734375, -21.052087783813477, -19.268037796020508, -43.212318420410156, 21.687185287475586, -43.2805061340332, 5.2694091796875, -29.137224197387695, -14.009174346923828, -9.497811317443848, -19.081087112426758, -36.520851135253906, 13.344581604003906, 1.0794353485107422, -38.578163146972656, 47.85212326049805, 39.494258880615234, -27.828489303588867, 40.263973236083984, 25.78026580810547, 7.480714321136475, -13.581154823303223, 0.11142310500144958, -5.478972911834717, 23.207294464111328, 21.905885696411133, -27.646852493286133, 0.9279041290283203, 9.52230167388916, 29.564720153808594, -14.29625415802002, -50.672637939453125, 41.29433822631836, 37.04669952392578, 22.996135711669922, 15.231751441955566, -17.741710662841797, -4.6397480964660645, -1.4358981847763062, 54.897552490234375, -8.411104202270508, 9.033675193786621, -16.994932174682617, 1.577566146850586, -0.21273016929626465, 31.430435180664062, 10.217937469482422, 58.19340133666992, 36.135494232177734, 15.074458122253418, 90.78019714355469, -25.0894718170166, 32.12068176269531, -2.9682421684265137, 58.080081939697266, -39.45271682739258, 5.896684646606445, -1.4670556783676147, -8.730632781982422, -30.678878784179688, 17.561420440673828, 93.32687377929688, -18.592201232910156, 56.32261276245117, 31.034542083740234, 31.091331481933594, -30.38004493713379, -39.898956298828125, -10.501677513122559, -19.120159149169922, -27.02416229248047, -15.32177734375, -43.62360763549805, 37.385433197021484, 13.85422134399414, -24.791887283325195, 16.746017456054688, -34.388797760009766, 24.174869537353516, 7.5167436599731445, 2.091167449951172, 32.7929801940918, 5.002950668334961, -56.016807556152344, -50.09056854248047, -26.810890197753906, -39.01789855957031, -63.30224609375, 30.368404388427734, -3.6684484481811523, -10.551322937011719, -48.8946418762207, 39.39474105834961, 35.39838790893555, 36.612274169921875, -9.952759742736816, -16.76825523376465, 32.214759826660156, -24.140779495239258, -13.308212280273438, -5.810249328613281, -4.807264804840088, -4.377265453338623, -4.5249176025390625, 1.6565608978271484, 27.731945037841797, -35.42039489746094, 45.02617263793945, 11.728772163391113, 5.254123687744141, -0.184970885515213, 24.704721450805664, 13.021576881408691, -53.00550079345703, -20.499500274658203, -13.532145500183105, -4.633849143981934, -15.145180702209473, 30.562088012695312, -36.33472442626953, 7.090147495269775, -23.9113826751709, -53.34759521484375, 88.5291748046875, -33.978858947753906, 22.659547805786133, 33.989017486572266, 50.771610260009766, -24.11617660522461, 21.024927139282227, 12.546806335449219, 13.738855361938477, 35.31754684448242, 19.733726501464844, 38.18027877807617, 64.51393127441406, 1.5060721635818481, 2.08939528465271, 11.00605583190918, -6.029325008392334, -29.85601043701172, 66.8748779296875, 47.42463302612305, 50.0296630859375, -74.1454086303711, -2.8184802532196045, 38.76321029663086, -4.861241340637207, -17.249834060668945, 34.71260452270508, -13.076606750488281, 2.230560064315796, 9.643254280090332, 18.76019287109375, -17.71637725830078, -19.102500915527344, -5.5507073402404785, -51.26425552368164, 8.305551528930664, -41.11015701293945, 54.97282028198242, -20.53938102722168, -29.632429122924805, -9.746686935424805, -46.016387939453125, -69.30608367919922, -42.25160598754883, -36.10114288330078, -27.414037704467773, 15.58665657043457, 6.691126346588135, -30.18662452697754, 37.55028533935547, 9.6847562789917, 50.79910659790039, 20.747282028198242, 67.30648803710938, -2.763932228088379, -10.443986892700195, -12.83703899383545, 9.728910446166992, 15.576549530029297, 15.518790245056152, 41.046295166015625, 1.663841962814331, 17.951309204101562, -25.113407135009766, -46.264469146728516, 22.810970306396484, -46.55710220336914, 19.37907600402832, 14.077399253845215, -48.63827896118164, 70.91600036621094, -11.560262680053711, 14.171727180480957, -42.927215576171875, 28.3161678314209, -35.26707077026367, -13.267922401428223, -4.7250566482543945, -14.766934394836426, 30.006532669067383, -7.588939189910889, 26.72923469543457, 44.21525192260742, 23.49949836730957, -10.674028396606445, -11.65932559967041, -17.354915618896484, -8.904057502746582, -31.066036224365234, 8.480108261108398, -18.664417266845703, 32.211910247802734, 27.611478805541992, 5.092136383056641, -45.06706619262695, 20.77875328063965, -34.93531799316406, -42.575199127197266, 13.867029190063477, 34.07368469238281, 9.100330352783203, -0.04305535554885864, -44.47795104980469, -12.764029502868652, 28.441354751586914, 47.5617790222168, -29.001605987548828, 42.94675827026367, 31.204370498657227, -7.366952419281006, -40.37378692626953, 5.402268886566162, -26.488651275634766, 20.117740631103516, -16.508039474487305, -66.03666687011719, 7.231614112854004, 3.401876449584961, 15.943991661071777, -11.639751434326172, 36.774356842041016, 31.706682205200195, 24.242191314697266, 5.562404632568359, 17.985427856445312, -47.596717834472656, 24.934818267822266, -8.835110664367676, -20.757225036621094, -28.24628257751465, -21.342636108398438, 59.76816940307617, 2.7199110984802246, 17.183683395385742, 72.82491302490234, -2.909379243850708, -83.66974639892578, 49.85148620605469, 67.40069580078125, -8.1627779006958, 33.15995407104492, 1.3644390106201172, -7.979040622711182, 33.37895584106445, -27.235055923461914, -51.03740310668945, 18.622304916381836, -1.2863883972167969, 6.585990905761719, 48.0951042175293, -10.970646858215332, -10.432096481323242, -89.83917999267578, -2.3662099838256836, 31.385692596435547, 33.00639343261719, -31.700807571411133, 11.940215110778809, -1.3966996669769287, -52.08514404296875, 24.387451171875, -49.56283187866211, -49.23586654663086, 11.226161003112793, -5.369723796844482, 12.164949417114258, 16.230607986450195, 10.66910171508789, 15.815994262695312, 20.35366439819336, 26.901805877685547, -2.703479051589966, -51.95205307006836, -20.906064987182617, 4.137994289398193, 17.04570770263672, 9.284594535827637, -10.175893783569336, -1.6180596351623535, -10.254228591918945, 14.3086576461792, 1.9201812744140625, 31.92466926574707, -42.3578987121582, 49.5699462890625, 29.895721435546875, -13.880523681640625, 24.90150260925293, -19.22768783569336, -39.724273681640625, -26.615062713623047, 30.52140235900879, -33.86849594116211, -47.9954719543457, 11.144562721252441, 13.460782051086426, -62.761138916015625, -1.7001386880874634, 8.281098365783691, 17.05070686340332, 26.187299728393555, 45.8919792175293, 0.3629723787307739, -17.579540252685547, 9.823963165283203, 39.3320426940918, -38.506103515625, 7.739323139190674, -18.17169761657715, 26.766870498657227, 26.40158462524414, 12.536096572875977, -24.678483963012695, -66.38018798828125, 4.2313385009765625, -9.427495956420898, 34.67371368408203, 13.61288070678711, 13.940850257873535, -5.120911598205566, 5.719011306762695, 40.74073028564453, 14.277429580688477, 9.38542652130127, 55.25260925292969, 0.342571496963501, 31.990203857421875, 12.95483112335205, 3.051030397415161, 2.002937078475952, -19.92416763305664, -14.320393562316895, 5.387230396270752, -4.382808208465576, -64.0718994140625, 37.93838882446289, 36.295806884765625, -72.32527160644531, -11.397019386291504, -9.411587715148926, -32.997310638427734, 38.61370849609375, -21.914371490478516, 30.919816970825195, -26.35954475402832, -24.112123489379883, -24.878976821899414, -8.67989444732666, -77.81048583984375, 29.480886459350586, 1.0324804782867432, -14.991109848022461, -6.543245315551758, -14.705904960632324, 3.1310782432556152, 56.57363510131836, 29.619653701782227, -38.506961822509766, 44.89359664916992, -49.639347076416016, -29.434885025024414, -35.14152908325195, 9.218317031860352, 1.5030202865600586, 27.429134368896484, -10.343448638916016, 28.610149383544922, -7.986376762390137, -4.891479015350342, -70.45719909667969, -6.966133117675781, -15.020964622497559, 11.499336242675781, 40.80420684814453, -39.09834289550781, 48.870941162109375, 16.31142807006836, -51.1062126159668, 33.855960845947266, 26.911285400390625, 15.122734069824219, -36.69544219970703, -79.67208862304688, 51.32204818725586, 5.019239902496338, -5.731456756591797, -46.96501922607422, -35.9199333190918, 7.173058032989502, 19.813230514526367, -45.07923126220703, 1.508231520652771, 12.56035327911377, 4.2415385246276855, 12.981339454650879, -21.763771057128906, -40.96165084838867, 34.113555908203125, 15.890563011169434, -38.586421966552734, 2.932729721069336, -15.755915641784668, -6.566326141357422, -1.1135469675064087, -22.95440673828125, -11.097309112548828, -22.465862274169922, -6.962340354919434, -17.42682647705078, 11.065922737121582, -18.36299705505371, -10.702925682067871, -27.52484703063965, -18.385421752929688, -89.79402923583984, -29.73658561706543, 6.119563579559326, 29.49297332763672, -21.535505294799805, 15.399569511413574, -39.6204833984375, 4.67535924911499, -32.72346496582031, 13.427494049072266, 20.1027889251709, 55.26424026489258, 35.63151550292969, -40.3484992980957, -1.7060450315475464, 15.293282508850098, -37.46352767944336, -40.58481216430664, 27.179784774780273, 0.8195980787277222, 25.622108459472656, 5.203431129455566, 36.41332244873047, -49.09945297241211, -17.089197158813477, 24.474300384521484, -8.257560729980469, 21.465944290161133, -22.389371871948242, -16.037731170654297, -6.7426276206970215, -8.598926544189453, -9.936942100524902, -34.673709869384766, -17.642337799072266, 11.917216300964355, 38.47869110107422, 42.290687561035156, -47.39043045043945, -35.697689056396484, 38.90782928466797, 30.132476806640625, 12.2085599899292, 28.69302749633789, -8.849040985107422, -6.092320442199707, -1.6220409870147705, 16.278522491455078, 1.7950432300567627, 26.209033966064453, 63.35904312133789, -67.30094909667969, 8.281928062438965, -7.548576354980469, 5.404679298400879, 28.947818756103516, -20.73760223388672, -17.69740867614746, 41.16762161254883, -53.92924499511719, 8.328821182250977, 23.936864852905273, 22.46142578125, 33.50288009643555, -36.01622009277344, -10.62491226196289, 80.18199157714844, -42.96570587158203, -19.97602653503418, 2.21340274810791, -39.574119567871094, -29.233949661254883, -18.173267364501953, 32.932186126708984, 22.02479362487793, 48.49467468261719, -44.54452896118164, -5.444217205047607, 14.194061279296875, 15.556614875793457, 55.99480056762695, 10.7008638381958, 2.187405586242676, 18.16347312927246, -8.576662063598633, 60.968902587890625, -13.722190856933594, -36.688018798828125, 3.5020575523376465, 6.568450927734375, -38.915035247802734, 5.901695251464844, -28.887357711791992, -46.91364288330078, 36.787750244140625, 1.5459332466125488, 5.038427829742432, 47.9744873046875, 8.299721717834473, 14.474039077758789, 19.636423110961914, 7.23432731628418, -45.118988037109375, 50.73638916015625, 17.803268432617188, 31.72987174987793, 5.368885040283203, -72.9832534790039, -49.571144104003906, 11.730345726013184, 89.83951568603516, 26.559856414794922, 0.056272029876708984, -18.84326934814453, 38.88924026489258, 11.294856071472168, 11.988645553588867, 27.224998474121094, -1.1498210430145264, 12.888029098510742, -24.849361419677734, -49.904754638671875, -44.602867126464844, -18.029300689697266, -0.9474364519119263, -41.326698303222656, 18.483936309814453, 54.4372673034668, 53.57606887817383, -20.79741096496582, 67.2431869506836, 16.266660690307617, -62.462093353271484, 61.28050994873047, 9.241827011108398, 22.556591033935547, -3.567948818206787, 40.88841247558594, 26.342666625976562, -10.249195098876953, 12.04751205444336, 8.4833345413208, 39.676876068115234, -6.373165607452393, 79.54014587402344, 8.954548835754395, -6.371415138244629, 51.07989501953125, 26.76437759399414, -1.610302448272705, -25.58523178100586, 1.6150257587432861, -11.682771682739258, 42.098411560058594, -19.07029151916504, -19.804054260253906, 9.922229766845703, -26.919103622436523, 29.64575958251953, -2.281329393386841, -10.623408317565918, -10.220657348632812, 26.542734146118164, -23.782093048095703, -6.51071834564209, -19.916881561279297, -40.49699401855469, 1.2441747188568115, -38.43747329711914, -45.9056396484375, 55.24027633666992, 21.099355697631836, -49.474632263183594, -10.330961227416992, -20.377492904663086, -9.926275253295898, -35.06207275390625, 12.484964370727539, 16.49207878112793, -25.18378448486328, -46.82364273071289, -31.093658447265625, 14.37436294555664, -4.183313846588135, 23.178447723388672, -8.158012390136719, 41.47116470336914, 35.06974792480469, -16.05603790283203, 22.0014591217041, -38.74455261230469, -63.534053802490234, -28.262645721435547, -2.648926258087158, 10.997690200805664, 40.09499740600586, 0.6241843700408936, -0.2960120439529419, -37.56013488769531, -27.201082229614258, -18.52513885498047, -32.03656005859375, -14.343611717224121, -43.14710998535156, 19.227262496948242, 0.24839353561401367, -24.897396087646484, -1.943763256072998, -16.894594192504883, 9.586913108825684, -6.95511531829834, -47.808780670166016, -2.950742483139038, -9.812199592590332, 10.636969566345215, 17.726871490478516, -32.98175811767578, -31.078554153442383, -9.339836120605469, -22.43507957458496, 7.966733455657959, 26.64014434814453, 5.618531703948975, -5.518409729003906, -5.166014194488525, -6.572854042053223, -35.452999114990234, 28.463878631591797, 14.048056602478027, 51.531700134277344, -35.14683151245117, 0.5389613509178162, 18.582229614257812, 14.800663948059082, -5.457082748413086, -37.9915657043457, -15.279861450195312, 17.153240203857422, 3.844761848449707, 26.65032958984375, -5.014172077178955, 6.212584972381592, 18.291521072387695, -0.936340868473053, 37.148433685302734, -40.44076156616211, -29.808727264404297, -3.0964841842651367, -38.40043258666992, 24.24350929260254, -18.93329429626465, 50.546897888183594, -8.591997146606445, 45.93325424194336, 24.433229446411133, -20.315753936767578, -49.115516662597656, 5.318456172943115, -19.761281967163086, -58.84675979614258, -25.761371612548828, 35.435523986816406, 10.702862739562988, 17.951204299926758, 30.150232315063477, -25.365863800048828, 27.661624908447266, -26.086360931396484, 19.392045974731445, 15.934530258178711, 5.152031898498535, 3.647103786468506, -25.1580753326416, 25.5911865234375, -51.680885314941406, 13.706635475158691, 44.64120101928711, 36.17412185668945, -10.575895309448242, -8.467605590820312, 21.389789581298828, 34.20004653930664, 16.006805419921875, -50.71243667602539, 44.9400634765625, -21.901960372924805, -76.60314178466797, -24.420331954956055, 6.340335369110107, 38.89271926879883, -28.327770233154297, -21.872007369995117, 23.208831787109375, 20.430028915405273, -5.317815780639648, 18.81548309326172, -22.619266510009766, -23.938892364501953, 13.92867374420166, 8.620916366577148, -10.929688453674316, 41.139129638671875, 27.40351104736328, -8.864750862121582, 30.96961212158203, -10.489400863647461, -8.136287689208984, -0.5331767201423645, -0.8740561008453369, -29.48965072631836, 24.544879913330078, 23.46030044555664, 8.88843059539795, -5.6326799392700195, 30.141071319580078, 1.9291636943817139, -40.72629165649414, -18.56995391845703, 33.77003479003906, -60.05770492553711, 50.70749282836914, -3.039462089538574, -36.62395095825195, 6.824779510498047, -44.233642578125, 1.8531873226165771, 56.917991638183594, 3.88494873046875, -17.679643630981445, -46.80473709106445, 25.442537307739258, -26.366960525512695, 55.06550216674805, 7.869102954864502, -3.223006010055542, -23.408138275146484, 63.068519592285156, -51.2205810546875, 54.44081115722656, 41.151893615722656, -18.53037452697754, 16.692861557006836, 47.06386184692383, 10.121875762939453, 13.416491508483887, 15.841653823852539, 21.75292205810547, -18.112709045410156, 8.652491569519043, 37.59426498413086, -4.7302045822143555, 13.211825370788574, 46.014373779296875, 23.65916633605957, -9.646309852600098, -24.329463958740234, -8.431058883666992, -8.653654098510742, -61.026763916015625, 65.3886947631836, 19.576414108276367, 4.512462615966797, -18.382061004638672, 0.00032508373260498047, 45.3869743347168, -1.5324864387512207, -14.553825378417969, 20.96322250366211, 23.04776954650879, 6.123084545135498, -40.231781005859375, -20.07318878173828, 28.507829666137695, -40.41386413574219, 46.2526969909668, 28.97115707397461, -7.202712059020996, -55.83721923828125, 15.908716201782227, 0.13983774185180664, 26.19394302368164, 18.923460006713867, 7.084959983825684, 25.904373168945312, -28.931198120117188, 5.274554252624512, 27.811935424804688, 12.184901237487793, -19.79941749572754, -12.163222312927246, -9.443263053894043, 31.460325241088867, -42.32752227783203, -20.257568359375, 31.273418426513672, -4.110602378845215, 7.127470970153809, -19.750099182128906, 4.226171016693115, 20.92426872253418, 20.559917449951172, -7.419729232788086, 38.47883224487305, 15.276686668395996, 33.505672454833984, 32.84437942504883, -21.987140655517578, -2.496253490447998, -21.17093276977539, -3.4332311153411865, 29.004667282104492, 13.036602020263672, 38.89785385131836, 31.635900497436523, -58.804168701171875, -3.1624526977539062, -19.717344284057617, -5.4489946365356445, -9.786139488220215, -38.07301330566406, -29.59956932067871, 16.930904388427734, -32.62186050415039, -13.175472259521484, 7.533420085906982, -43.04228210449219, 11.479653358459473, 61.23909378051758, 3.458305597305298, -20.04507064819336, 44.18693542480469, 15.62531566619873, -5.389920234680176, -50.136077880859375, 21.684850692749023, 12.525307655334473, -19.74645233154297, 46.92190933227539, -31.63050079345703, 24.301441192626953, 16.76988983154297, 10.333394050598145, 37.616485595703125, 14.429634094238281, -19.656272888183594, 18.34342384338379, -16.17593002319336, 4.3782243728637695, 19.08709716796875, 0.8877235651016235, 30.043500900268555, 94.69839477539062, -45.48585891723633, 10.109407424926758, 4.883814334869385, 20.56415557861328, 38.78562545776367, 40.65460205078125, 28.10686683654785, -33.457603454589844, -17.84009552001953, -7.657515048980713, 34.53973388671875, -9.692514419555664, -41.92182159423828, -10.345989227294922, 43.803348541259766, 25.1953182220459, -7.986118793487549, 20.680089950561523, -18.130420684814453, -58.6673469543457, 23.56127166748047, 33.57151412963867, 14.504606246948242, -21.40235137939453, 36.3857307434082, -11.387452125549316, 25.15856170654297, 28.53569793701172, -19.07772445678711, -27.47899055480957, 26.376516342163086, -11.493071556091309, -21.156192779541016, -47.210655212402344, -10.513924598693848, 19.773813247680664, 14.903682708740234, 31.19672966003418, -26.402912139892578, 18.03868865966797, 42.9954833984375, -14.060867309570312, 15.89514446258545, -9.56666374206543, 41.46611785888672, -9.235377311706543, -3.3380143642425537, -12.994958877563477, 3.8516547679901123, -10.72784423828125, -11.855826377868652, -1.9596970081329346, 23.8602294921875, -11.367398262023926, -0.53208327293396, 21.65268898010254, -6.108818054199219, 5.392580509185791, 17.21846580505371, 33.79932403564453, 23.170801162719727, -2.10198974609375, -36.14094161987305, -38.04285430908203, 9.784648895263672, 16.21593475341797, -0.02879035472869873, -11.256930351257324, 0.22924983501434326, -44.97418975830078, 43.71651077270508, -17.202260971069336, 31.467844009399414, 3.113262176513672, -7.0740132331848145, -56.97931671142578, -26.945396423339844, 34.9437141418457, 12.103619575500488, -27.581707000732422, 35.2352409362793, 0.6734843850135803, -17.216806411743164, 33.08359909057617, 58.4589958190918, -39.9990348815918, -18.10847282409668, 15.648087501525879, 43.53053283691406, 22.989059448242188, 27.33843421936035, -58.25367736816406, -32.86529541015625, 10.79525089263916, -5.3947649002075195, -58.95700454711914, 4.6823530197143555, -24.68202018737793, 18.163684844970703, 10.643241882324219, 30.711368560791016, 20.161298751831055, -30.80220603942871, -16.084796905517578, 32.26905059814453, 47.335479736328125, 28.625415802001953, -2.881349563598633, 26.57418441772461, 25.620800018310547, -3.0617012977600098, -12.593235969543457, -1.7074874639511108, 22.97793960571289, -18.186553955078125], [29.517385482788086, 16.53900718688965, -21.657312393188477, -27.095651626586914, 36.12748336791992, -38.83871841430664, 20.30038833618164, -27.002588272094727, 65.12484741210938, -32.67677307128906, -54.92795181274414, 43.653507232666016, -16.40487289428711, 48.58127975463867, -26.305171966552734, -9.078058242797852, -20.22507095336914, -27.560199737548828, 34.032981872558594, 62.26579284667969, 45.68113327026367, 46.7067756652832, -12.268348693847656, -27.790842056274414, 34.307533264160156, -17.62246322631836, 8.236050605773926, 36.88467025756836, -8.662093162536621, 15.994013786315918, -37.10112762451172, 79.23011779785156, 18.49476432800293, 25.373409271240234, -24.02777099609375, 38.480712890625, -31.974266052246094, 42.08136749267578, 16.29770278930664, -17.941953659057617, 36.85093307495117, 0.4953976273536682, -4.283864498138428, 36.29038619995117, -7.8332085609436035, -9.902727127075195, 14.219524383544922, 29.28030776977539, -58.43916702270508, 28.964218139648438, -35.81805419921875, -34.018680572509766, -3.9344866275787354, 15.667572021484375, 40.428062438964844, -9.453611373901367, -42.88850021362305, 45.16044998168945, 8.70375919342041, -20.147401809692383, -10.817557334899902, -72.13153839111328, -10.442743301391602, 70.74532318115234, -38.26292037963867, -28.2698974609375, -48.443702697753906, -5.1161885261535645, 3.9976425170898438, 59.72440719604492, 22.54294776916504, 13.09561824798584, -13.211772918701172, 3.4794328212738037, 3.611891746520996, -44.646392822265625, -40.942073822021484, 28.86126136779785, -15.800289154052734, 10.322060585021973, -7.191989421844482, -18.109270095825195, 12.64669418334961, 7.030245304107666, 2.394648551940918, -72.44757843017578, 40.13798522949219, -7.205057144165039, 25.404373168945312, 12.20622730255127, -42.180641174316406, 21.845643997192383, 96.31734466552734, -9.015862464904785, 19.073013305664062, -49.38170623779297, -17.46381378173828, 40.070167541503906, -3.053131580352783, 1.0072052478790283, 48.47332000732422, -23.111574172973633, -19.997583389282227, 6.162388801574707, 16.085498809814453, 23.186880111694336, 1.2761425971984863, 44.67879867553711, -80.81082916259766, -50.54973220825195, 21.598440170288086, -22.424070358276367, 34.781436920166016, -27.931066513061523, 26.093461990356445, -6.047971248626709, -9.285849571228027, -2.8558807373046875, -8.315901756286621, -9.26124095916748, -4.487643718719482, -21.402851104736328, -18.047412872314453, 24.811614990234375, 11.346000671386719, -38.837528228759766, -26.745025634765625, 29.100486755371094, -3.239650011062622, -25.822795867919922, 23.682453155517578, -68.13004302978516, 28.278181076049805, 8.118617057800293, -6.188776016235352, -36.25243377685547, 15.604415893554688, -29.616104125976562, 7.732786655426025, 18.861650466918945, 20.733901977539062, -6.759131908416748, -31.431859970092773, -1.0393013954162598, 48.55113983154297, -8.540038108825684, 12.62828540802002, 9.551698684692383, -113.74610900878906, 18.852989196777344, -16.369770050048828, -32.91367721557617, 9.159358024597168, -3.027155876159668, -65.9962387084961, 63.5289306640625, 33.94908905029297, -17.781171798706055, 14.393548965454102, 57.661258697509766, -0.00504148006439209, 4.808488368988037, 16.015352249145508, 13.353126525878906, -20.005998611450195, 40.29393005371094, -16.030134201049805, -5.815299987792969, -41.2180061340332, 3.5041286945343018, -1.1313951015472412, 38.26346206665039, -24.395734786987305, 0.5456430912017822, 18.23651885986328, 25.192293167114258, 15.813456535339355, 2.9263010025024414, -64.92344665527344, -59.26476287841797, 33.63481521606445, -21.20278549194336, -21.512813568115234, 7.772477149963379, -31.45355796813965, -41.0563850402832, 17.509565353393555, -26.458602905273438, -70.90410614013672, 33.748748779296875, 21.211872100830078, -16.394439697265625, -1.8237650394439697, -16.753707885742188, 15.698451042175293, -25.757312774658203, -32.46002197265625, -22.135595321655273, -48.64064407348633, 53.875396728515625, -39.93868637084961, 4.032517910003662, 28.178926467895508, -19.310401916503906, -5.720263957977295, 9.213482856750488, 27.138351440429688, -13.289892196655273, -37.56214904785156, -21.55370330810547, -17.837894439697266, -10.139847755432129, 14.17885684967041, -19.19020652770996, 26.63129234313965, 19.926589965820312, 12.24007797241211, 17.142650604248047, -27.57947540283203, -25.992158889770508, 14.504006385803223, -10.290975570678711, -4.240777015686035, 20.809282302856445, -21.779041290283203, -51.1038932800293, 28.251907348632812, -7.07180118560791, -26.303913116455078, 27.673004150390625, 10.073487281799316, -30.677244186401367, 20.380008697509766, 41.299007415771484, -11.556897163391113, 68.91468048095703, -2.014735221862793, -16.76814842224121, 22.098085403442383, -8.901147842407227, -14.58049201965332, 14.324934959411621, -3.723984718322754, 33.01709747314453, -11.165682792663574, -23.45358657836914, 36.12336730957031, 15.869134902954102, 34.49435806274414, 18.053199768066406, 9.959522247314453, 11.001731872558594, 25.59661293029785, 28.180950164794922, 32.54832077026367, -7.071433067321777, 8.286520004272461, -4.9242072105407715, -13.382233619689941, -17.989669799804688, 64.84194946289062, 20.516202926635742, -11.41358757019043, -37.97421646118164, 9.900137901306152, 5.801571369171143, -35.90259552001953, 0.9496594071388245, 13.406343460083008, -27.234027862548828, -102.5073471069336, -23.68638038635254, 27.393848419189453, -10.225654602050781, -14.602725982666016, -10.55927562713623, -30.73101043701172, -39.542423248291016, 19.788454055786133, -8.982881546020508, 6.292875289916992, -29.78919792175293, -29.646076202392578, 7.286892414093018, 29.73480796813965, -44.0881233215332, -26.210851669311523, -4.529419422149658, -32.53412628173828, -9.150978088378906, 20.783571243286133, -11.633687019348145, 8.245615005493164, 40.57908248901367, 20.886350631713867, 0.5134969353675842, 43.7449951171875, -36.438941955566406, 16.12676239013672, -15.841432571411133, -57.36643981933594, 6.01351261138916, 34.734100341796875, -15.542673110961914, -19.201128005981445, -13.7894926071167, 10.934163093566895, 41.19783401489258, 63.6262321472168, 11.887617111206055, 9.379426002502441, 22.16496467590332, -5.407044410705566, 39.94155502319336, 2.334221363067627, -15.028762817382812, 23.52073097229004, -31.609561920166016, -57.95669937133789, 15.110417366027832, -4.457409381866455, 52.394386291503906, -6.189935684204102, -26.907062530517578, -25.395401000976562, 37.17841720581055, 16.816923141479492, 23.754106521606445, 10.866127014160156, -16.5489444732666, 6.1505584716796875, 32.43629837036133, -1.4784979820251465, -41.60002899169922, -26.16875457763672, 34.07203674316406, -6.930820941925049, 13.549762725830078, -21.01506996154785, 54.217689514160156, -5.970200538635254, 29.15896987915039, 6.091814041137695, -2.7149276733398438, -9.012232780456543, -22.73999786376953, 10.222002029418945, -5.45477819442749, 14.816944122314453, -28.7397518157959, 28.28757667541504, -15.673476219177246, 20.228897094726562, 0.8328638076782227, -25.885353088378906, 32.357357025146484, 1.1565834283828735, 29.109201431274414, 43.18925476074219, 3.672741174697876, 3.344146966934204, 34.61838150024414, 55.03611755371094, -9.163216590881348, 23.823974609375, -32.02815246582031, -3.507582902908325, -0.4322929382324219, 34.083091735839844, -57.677528381347656, -10.090152740478516, -18.648454666137695, -39.88368606567383, -21.228290557861328, 20.1925106048584, -5.809000492095947, -23.787656784057617, -6.546703338623047, -8.997888565063477, -8.439925193786621, -43.141422271728516, 84.61264038085938, -44.96051025390625, 30.516815185546875, 29.935394287109375, -26.250442504882812, -4.336568832397461, 14.037250518798828, 11.640108108520508, 9.704298973083496, 19.698823928833008, 9.688691139221191, -2.7641220092773438, 8.079010009765625, 22.62706756591797, -0.9001957774162292, 0.01307374332100153, -18.185977935791016, -14.408093452453613, -16.581314086914062, -36.98440170288086, 10.580411911010742, -16.986347198486328, -30.604909896850586, 14.528461456298828, 10.683218955993652, -32.81066131591797, 18.39410400390625, 50.83528137207031, -1.5700018405914307, -17.82720184326172, 50.61588668823242, -6.00210428237915, -21.566280364990234, 12.148571014404297, -18.839420318603516, -54.80894470214844, 37.850830078125, -22.00062370300293, 36.253173828125, 14.988176345825195, -19.46949577331543, -4.483338356018066, 21.998701095581055, 37.90841293334961, -45.165157318115234, -55.51649856567383, -24.92047691345215, -66.75590515136719, 13.333440780639648, 12.732885360717773, -21.243030548095703, 23.77452278137207, -23.71725082397461, -50.5125617980957, 21.29009246826172, 5.393754959106445, 25.054866790771484, -0.34159988164901733, 9.707358360290527, -5.426682472229004, 49.121795654296875, 14.39153003692627, -34.348358154296875, -14.291807174682617, 27.923851013183594, 5.3468241691589355, -51.14198303222656, -47.74595642089844, -23.71728515625, 3.001803159713745, -22.451133728027344, -10.483421325683594, 11.101606369018555, -51.26613235473633, -38.01735305786133, -22.50426483154297, 32.80155563354492, -3.2253127098083496, -22.224773406982422, -3.3896491527557373, 34.407073974609375, 4.863947868347168, 15.200871467590332, -13.287393569946289, 6.3671369552612305, -16.199506759643555, 14.04718017578125, 2.4403014183044434, -6.1532721519470215, 3.629554510116577, 39.12856674194336, 2.328801393508911, 24.256196975708008, -24.172351837158203, -29.785812377929688, -33.81692123413086, -38.49205780029297, 0.32176971435546875, 49.50485610961914, 22.584630966186523, 2.601921558380127, -20.5070858001709, -36.69740295410156, -38.34258270263672, -22.90768051147461, 24.60824966430664, -12.85318660736084, 18.90553855895996, -53.03951644897461, 17.63457489013672, 7.636490345001221, 65.58722686767578, 32.51033020019531, -49.098873138427734, 13.171833038330078, 9.668277740478516, -17.607181549072266, 53.871978759765625, -38.057247161865234, 31.236446380615234, -29.905630111694336, 53.415550231933594, 63.10633850097656, -3.7274999618530273, 2.0118906497955322, -3.6167843341827393, -0.5069215297698975, 33.0240592956543, 22.657682418823242, -42.69275665283203, -0.06785786151885986, 10.174112319946289, -13.642461776733398, -6.897593975067139, -8.647388458251953, 38.5140380859375, -13.679059028625488, -28.144779205322266, 21.821821212768555, 18.452970504760742, -5.154404640197754, -4.214359760284424, 28.97620964050293, -13.8845796585083, 11.727025032043457, -38.36796951293945, -23.590059280395508, 2.20086932182312, -40.20953369140625, 27.855905532836914, -23.178422927856445, -17.793027877807617, 35.55801773071289, 15.793128967285156, -21.423938751220703, -17.705259323120117, -29.933887481689453, -42.258792877197266, 10.071874618530273, -68.3851089477539, -2.6518807411193848, 74.51783752441406, 1.2593849897384644, -2.8855297565460205, -0.11562752723693848, -38.108028411865234, 47.87952423095703, 3.971376895904541, 9.730161666870117, 20.67734146118164, -26.51195526123047, 33.4445915222168, 6.773116111755371, 34.67301559448242, -14.0156888961792, -53.71205520629883, 13.08080005645752, -15.80284595489502, 8.327788352966309, 41.857967376708984, 20.9735164642334, 23.42359733581543, -3.275869846343994, 36.79483413696289, -13.669267654418945, 3.322725772857666, -19.205961227416992, 9.209543228149414, -19.72012710571289, 38.3168830871582, -86.4997787475586, -58.2234001159668, 44.92816162109375, 21.389297485351562, 26.152963638305664, -54.83478927612305, 47.3005485534668, -48.53044128417969, 14.099946022033691, -8.321667671203613, 14.643016815185547, -34.93050003051758, 41.785343170166016, -14.987032890319824, 37.53330993652344, 41.528038024902344, -21.65602684020996, -15.196191787719727, 18.038206100463867, 12.511219024658203, 19.53376007080078, -3.3326354026794434, -30.55718421936035, 12.951873779296875, 9.653182029724121, -39.87663650512695, 68.9495849609375, 5.832306385040283, 3.7986507415771484, -33.99210739135742, 26.223224639892578, -13.142363548278809, 21.921775817871094, -23.381099700927734, -16.113473892211914, -43.340553283691406, -57.22835159301758, -22.270030975341797, 30.66867446899414, -20.855833053588867, 23.857730865478516, 49.607791900634766, -31.80240249633789, -18.270112991333008, -36.67424392700195, -13.856051445007324, 3.247253894805908, 2.563204526901245, -38.297977447509766, -25.009279251098633, -0.5826976299285889, 21.107501983642578, -6.446667194366455, 34.64894485473633, -6.380502700805664, 56.237857818603516, 24.62140655517578, -9.43488597869873, -43.94676971435547, -23.060495376586914, -30.38447380065918, -14.954191207885742, -9.965873718261719, -22.959312438964844, -13.386173248291016, 35.432559967041016, 1.3675545454025269, 39.47217559814453, 6.392209529876709, -14.458948135375977, -7.06876802444458, -0.5804556608200073, -12.261801719665527, 21.374061584472656, 7.7953643798828125, -29.31494140625, 5.580477237701416, 26.989681243896484, -29.26789093017578, -4.069179058074951, -10.42142391204834, -27.855131149291992, -4.908779621124268, 4.879035472869873, -15.562990188598633, -23.878068923950195, 3.8442111015319824, -19.02223777770996, 33.25750732421875, 42.85518264770508, -29.937419891357422, -11.004322052001953, -3.2966725826263428, -24.079866409301758, 20.986650466918945, 28.17544174194336, 0.27492260932922363, -7.312061786651611, -30.649417877197266, 19.91332244873047, 30.962787628173828, -1.0639735460281372, 15.054725646972656, 52.52781677246094, 38.580204010009766, 45.840171813964844, -47.702701568603516, 29.005752563476562, 7.155807018280029, -14.577692031860352, -22.452861785888672, -32.843326568603516, 18.610380172729492, 12.443867683410645, -2.9256534576416016, -17.1450252532959, 10.54079532623291, -33.51008605957031, 4.128276348114014, 31.170900344848633, 27.85662841796875, -19.21440887451172, 18.52791976928711, 5.87734317779541, 66.5295181274414, -41.06443405151367, -19.624420166015625, -6.659925937652588, -46.33106231689453, -2.3208682537078857, 50.476966857910156, -13.405869483947754, -19.952068328857422, -37.21627426147461, -3.6628506183624268, -43.64211654663086, -27.17051887512207, -5.666489601135254, -15.576672554016113, 39.59254837036133, 73.2117691040039, -25.262788772583008, 50.705257415771484, 7.939370155334473, 9.717277526855469, 1.192701816558838, 27.55512046813965, -56.72935104370117, -1.4746829271316528, -5.197325229644775, 4.167191982269287, -59.29397964477539, -9.27686882019043, -7.046543598175049, -52.89975357055664, -23.674564361572266, 13.113286972045898, 40.57815170288086, 1.4588261842727661, 37.6538200378418, -6.985361099243164, -20.344497680664062, -47.028602600097656, -33.033119201660156, 3.503204822540283, 15.642661094665527, -52.18819046020508, 39.28750991821289, 62.870723724365234, -11.29150676727295, 53.328346252441406, -15.703325271606445, -18.44469451904297, -16.68250274658203, -47.99788284301758, -38.532928466796875, 27.47828483581543, 37.5074462890625, -32.01664352416992, -26.744901657104492, -2.918081760406494, -86.90070343017578, 18.400869369506836, 10.91313362121582, 3.343444347381592, -0.4453926682472229, -22.94989013671875, -13.101819038391113, -6.279101371765137, 5.764348030090332, -19.853435516357422, 17.0825138092041, 29.53867530822754, -15.271748542785645, 26.05152702331543, 22.175729751586914, -29.53424072265625, 5.716754913330078, 34.676612854003906, -56.33573913574219, -62.64238357543945, -10.00544548034668, 0.22666728496551514, 41.816585540771484, -8.135945320129395, 19.116546630859375, -13.478524208068848, -5.742809772491455, -58.90769577026367, 7.20379638671875, 81.67639923095703, 11.45672607421875, -8.746358871459961, 5.490435600280762, -37.68115234375, 22.665555953979492, -67.74259185791016, -16.69565200805664, -11.421903610229492, 6.498039722442627, -35.911399841308594, -8.223365783691406, -7.05741024017334, 10.895895957946777, 7.802262306213379, 10.624192237854004, 34.457828521728516, 3.6929454803466797, -2.421393394470215, 65.31816101074219, 21.04490089416504, 50.69575119018555, 2.2105095386505127, 29.067201614379883, -45.025577545166016, 17.78165626525879, 7.418344497680664, 1.9976822137832642, -18.543996810913086, 6.845637798309326, 6.943083763122559, 14.020856857299805, 10.551968574523926, -7.669739723205566, -13.174437522888184, -19.166284561157227, -39.72196578979492, -7.941160202026367, -33.60004806518555, -21.202919006347656, 6.050906658172607, -5.330530643463135, -8.376335144042969, 7.672055721282959, 21.124685287475586, 10.864548683166504, -5.768951416015625, -13.380155563354492, 15.735867500305176, -11.066987037658691, 9.34592342376709, 21.116771697998047, 37.05646514892578, 30.448514938354492, -25.87290382385254, -29.70002555847168, 12.946111679077148, -21.440122604370117, -5.153298377990723, -12.825496673583984, -14.86963176727295, -34.65306091308594, 15.578465461730957, 17.3834285736084, -22.46988868713379, -43.966285705566406, 3.4270288944244385, 29.70266342163086, 16.698522567749023, -22.64920997619629, 8.036696434020996, -0.7359110713005066, 1.0828460454940796, 19.797300338745117, 1.682376742362976, -26.70640754699707, 48.197021484375, 37.365474700927734, -44.214996337890625, 7.4129252433776855, -6.834372043609619, 0.8097001910209656, -25.748981475830078, 33.332862854003906, -7.102524757385254, 38.59561538696289, 58.8179817199707, -13.811250686645508, 39.637611389160156, 33.17010498046875, -27.89112091064453, 6.699452877044678, 16.63539695739746, -11.441570281982422, -14.679037094116211, -0.691436767578125, -12.731511116027832, -27.24683380126953, -19.215909957885742, -5.14439058303833, 19.356355667114258, 20.96942901611328, -13.275285720825195, 65.97801208496094, 12.106058120727539, 9.876533508300781, 5.576700210571289, 28.577980041503906, 34.4990119934082, -7.789167404174805, -18.520498275756836, 7.988636016845703, 18.277162551879883, 37.58631134033203, -35.86299133300781, 31.80732536315918, -5.213494300842285, -19.647653579711914, 17.2033748626709, 13.994111061096191, -28.91396141052246, 7.973217487335205, -2.7725627422332764, 10.17403793334961, 9.856016159057617, -37.52363967895508, 16.667905807495117, 38.35820770263672, 5.235750198364258, -29.8059024810791, 100.97389221191406, -35.65467071533203, 1.0012438297271729, 3.2977983951568604, -17.48173713684082, -28.784879684448242, 26.674165725708008, -39.980106353759766, 16.980497360229492, -4.34476375579834, 12.302841186523438, 7.1756486892700195, -16.584440231323242, 24.872459411621094, -35.87256622314453, 48.5493049621582, 4.005776882171631, -6.032829761505127, 18.916715621948242, 2.9494268894195557, -26.64035987854004, 32.79583740234375, 11.648679733276367, 12.03123950958252, -8.00538444519043, 39.88345718383789, -11.40770149230957, -32.877628326416016, 8.77092170715332, 18.881567001342773, 42.78048324584961, -14.345585823059082, -23.34486198425293, 5.483965873718262, 46.614105224609375, -30.581737518310547, 40.712379455566406, -10.433591842651367, -8.61799430847168, -55.88536071777344, -6.739012718200684, -32.989280700683594, -3.5925443172454834, 82.12370300292969, -20.433969497680664, -27.482824325561523, 31.14793586730957, 5.733776092529297, -21.78628158569336, -73.28070831298828, -11.919246673583984, -53.53990936279297, -15.391014099121094, 17.230873107910156, -16.067115783691406, -89.5057601928711, -7.609997749328613, -9.836128234863281, -6.9570112228393555, -13.246048927307129, -1.500655174255371, 47.77372741699219, 21.036314010620117, -2.186455249786377, -29.938491821289062, -27.23402976989746, -51.583866119384766, -23.729625701904297, -5.484719753265381, 8.411636352539062, -3.4350852966308594, 28.44937515258789, -37.08652877807617, -25.84200096130371, -13.566560745239258, 3.0116326808929443, -3.990968704223633, 37.066219329833984, 44.959266662597656, -20.03657341003418, -23.762683868408203, 23.49344825744629, -53.75251388549805, -10.366381645202637, -29.154802322387695, 6.338452339172363, 56.331661224365234, 1.9569199085235596, 1.3232083320617676, 29.191211700439453, 31.141490936279297, -31.393535614013672, 7.569174289703369, 23.820585250854492, 4.14448356628418, -1.916959285736084, -6.150633811950684, 5.4046854972839355, 64.67018127441406, 9.147873878479004, 33.67259216308594, 59.32637023925781, 42.21501159667969, 43.17927932739258, 4.5934343338012695, 14.598012924194336, -54.56527328491211, -34.15791320800781, 13.300745964050293, -42.35358810424805, -61.85222244262695, -41.11453628540039, 42.44351577758789, 21.63898277282715, 3.043372392654419, 18.011070251464844, 2.109684467315674, -35.16572952270508, -12.894322395324707, -17.511627197265625, -3.02107310295105, -1.12755286693573, -2.7988970279693604, 1.9342135190963745, -28.080286026000977, -14.089313507080078, 48.30525588989258, 12.659361839294434, 11.722936630249023, -3.5736849308013916, -9.106929779052734, 7.066468715667725, -16.662395477294922, -2.2645721435546875, 49.66943359375, 29.0273494720459, 2.60970139503479, -30.654998779296875, -15.08584976196289, -11.540409088134766, 24.51711654663086, 18.16777992248535, 39.38703536987305, 28.335405349731445, -18.842226028442383, -1.9383625984191895, 0.43012964725494385, -2.9359498023986816, -15.656332015991211, 40.7337646484375, -3.174980878829956, 52.0828971862793, -13.335684776306152, 6.209702968597412, 20.409456253051758, 22.651960372924805, -11.983062744140625, -32.151092529296875, 15.476788520812988, 7.0558695793151855, -51.47991180419922, -31.392560958862305, -11.612619400024414, -0.3646603226661682, 49.89580154418945, -46.363731384277344, -10.758081436157227, 3.4468531608581543, -83.77310180664062, -40.26519775390625, 43.889305114746094, -25.13858985900879, -3.2109079360961914, -15.252192497253418, -41.30107498168945, -11.12197494506836, 23.221630096435547, 27.43222427368164, -29.97488784790039, 58.300926208496094, -26.608064651489258, -28.25668716430664, 3.67875337600708, 38.694366455078125, 3.1935720443725586, -20.69132423400879, 43.50152587890625, 0.17610874772071838, -35.9992561340332, -14.819189071655273, 12.588597297668457, -6.450310707092285, 53.00182342529297, -15.506061553955078, -3.297607421875, 8.517440795898438, 4.424243450164795, 19.1273250579834, 13.30835247039795, -22.768461227416992, 47.307682037353516, -6.314195156097412, -19.776105880737305, -2.194669723510742, -60.86861038208008, 47.653594970703125, -30.137939453125, -9.929708480834961, -38.59795379638672, -4.259250640869141, 10.825899124145508, 40.813228607177734, 7.519469261169434, -23.799541473388672, -14.272948265075684, 30.98309326171875, 2.778155565261841, -11.609334945678711, -39.59364318847656, 10.587438583374023, -5.948300838470459, 9.159957885742188, -4.174489974975586, -12.099092483520508, 12.69755744934082, -34.385780334472656, 34.0220832824707, 2.381032943725586, 2.875920295715332, 59.56376647949219, -27.29107666015625, 7.3074259757995605, -6.676034927368164, 7.63674783706665, 6.050418376922607, -6.454190254211426, -2.9415040016174316, -22.679046630859375, 46.10276794433594, -10.494278907775879, 6.851271152496338, -27.241350173950195, 45.632450103759766, 47.77445602416992, 8.481605529785156, 9.578082084655762, -24.71895980834961, -44.76871871948242, -27.020463943481445, -21.53803253173828, 35.9998893737793, -44.59968566894531, 32.53343963623047, -13.13525676727295, 19.865280151367188, -20.34178924560547, -16.006755828857422, -14.016640663146973, 9.853022575378418, 4.800292015075684, 14.469856262207031, -35.590301513671875, -22.280136108398438, 29.409969329833984, 26.24445915222168, -14.713617324829102, 3.1892361640930176, -42.697540283203125, 8.102221488952637, -15.590447425842285, 23.640417098999023, 52.94678497314453, -31.95590591430664, 23.017852783203125, 46.87185287475586, -72.16920471191406, -18.40770149230957, -6.692519187927246, 18.7138614654541, 14.785426139831543, 17.049158096313477, -54.846275329589844, 28.426298141479492, 5.989823818206787, 58.7437744140625, -16.44770050048828, -11.397571563720703, -35.17038345336914, 12.693927764892578, 9.170708656311035, 69.33438110351562, -48.7088623046875, 43.52798080444336, 17.529224395751953, -22.804889678955078, 26.94757843017578, 22.700273513793945, 7.527763366699219, -3.5226004123687744, -0.4635113477706909, -12.414344787597656, -44.10942077636719, -25.843721389770508, -2.045323610305786, -3.3972983360290527, -30.321460723876953, 16.359376907348633, 33.98080062866211, 42.443450927734375, -2.996633291244507, -39.524574279785156, -6.703784465789795, 15.328703880310059, 34.36012649536133, 31.279266357421875, -50.29149627685547, -23.90787124633789, -19.30358123779297, 7.76952600479126, -25.906005859375, -22.36690330505371, -7.3006415367126465, -41.52544021606445, -10.717766761779785, 34.46025085449219, -39.01536560058594, -22.15334129333496, 21.834230422973633, -29.63813591003418, 37.83390426635742, -78.55712127685547, 31.163400650024414, -9.792062759399414, 60.516334533691406, 10.423148155212402, 37.649024963378906, 18.903640747070312, -16.145709991455078, -48.12516403198242, -26.466148376464844, 32.65174865722656, -37.88088607788086, -37.58506393432617, -50.56877517700195, 52.9812126159668, -0.508615255355835, 28.929655075073242, 9.155939102172852, -9.830862998962402, -36.408966064453125, -5.2157793045043945, 69.29788970947266, -0.2953415513038635, 19.195823669433594, 4.498611927032471, 11.89527702331543, -4.6819658279418945, -19.34238052368164, 43.301326751708984, -9.982572555541992, 29.413972854614258, 62.32122802734375, -10.07986068725586, 13.379459381103516, 34.17074203491211, -12.763477325439453, 58.239139556884766, -21.931880950927734, -14.728765487670898, 23.257041931152344, 4.129556655883789, 30.862930297851562, -20.775875091552734, 12.943293571472168, 29.552127838134766, -19.661548614501953, 28.123397827148438, -23.367815017700195, 48.92563247680664, -20.537059783935547, -12.6104154586792, 3.6811330318450928, 4.84660005569458, 7.957382678985596, 7.796139717102051, 13.096334457397461, 0.9789590835571289, 15.183714866638184, 17.181913375854492, 25.606822967529297, 5.87544059753418, 1.4231129884719849, 14.160468101501465, -50.31315612792969, 21.511550903320312, -34.541908264160156, -8.696340560913086, 4.117852210998535, -27.611717224121094, 8.402810096740723, -3.836221218109131, 15.454946517944336, -31.821170806884766, -17.28264808654785, 8.259528160095215, -38.55034255981445, 4.913157939910889, 30.620393753051758, 68.09503173828125, -1.9602534770965576, -11.511503219604492, 38.89985275268555, 44.91761016845703, -41.401153564453125, 4.654221534729004, 14.063807487487793, -44.19009017944336, 27.711299896240234, 18.294979095458984, 5.485572338104248, 5.900056838989258, 45.54724884033203, -35.56307601928711, -38.48542022705078, 19.917434692382812, 32.67809295654297, -4.984264373779297, -7.610029220581055, -80.36164855957031, 3.3534135818481445, 51.93052673339844, -14.74563217163086, 32.9608039855957, 75.95804595947266, -12.402911186218262, -24.084884643554688, -22.376577377319336, 55.57667541503906, 5.417474746704102, 26.85320281982422, -5.477538585662842, 28.27406120300293, 37.67300796508789, -18.69034194946289, 15.040006637573242, -10.196781158447266, -11.852486610412598, -14.609048843383789, -22.106014251708984, 10.379711151123047, -7.595338821411133, 18.60053062438965, -26.694358825683594, 35.9379997253418, -19.686574935913086, -38.073570251464844, 4.238149642944336, -50.15142822265625, 35.93090057373047, -62.333683013916016, 80.2154312133789, 12.780420303344727, 27.030498504638672, -11.080537796020508, 17.5137939453125, 4.064872741699219, -43.62976837158203, 29.239713668823242, -9.483968734741211, 14.234077453613281, -23.81744956970215, -13.144637107849121, 6.644059181213379, 35.701236724853516, 16.575023651123047, 11.868228912353516, 18.90218162536621, -27.844762802124023, 30.2854061126709, -14.595799446105957, -69.05757141113281, -71.12300109863281, -17.62040901184082, -2.392533302307129, 34.44929885864258, 1.255523681640625, -13.625561714172363, -25.05738639831543, 47.24433517456055, -12.59953784942627, 7.1683807373046875, -36.779659271240234, -23.800918579101562, 24.627113342285156, -14.367530822753906, 54.81180191040039, 30.900123596191406, -14.67957878112793, 20.502239227294922, -6.507565498352051, 28.197601318359375, 20.64803123474121, 26.163434982299805, 38.950443267822266, -5.209863662719727, -28.867313385009766, -9.734051704406738, 6.753161907196045, -26.48885154724121, 9.39551830291748, -15.168346405029297, 29.3145694732666, 3.421757459640503, 4.395389556884766, 7.877647399902344, -29.44953727722168, 17.444263458251953, -1.2911795377731323, -0.5972563028335571, -1.0446124076843262, 1.978596806526184, -64.49674987792969, -23.470077514648438, -49.411319732666016, -0.8126012086868286, -26.191423416137695, -22.857994079589844, -2.392451763153076, -29.084447860717773, 40.291038513183594, -3.560142755508423, -9.435142517089844, 36.92462158203125, 46.38700866699219, 11.406109809875488, 22.725555419921875, -11.758216857910156, 12.789652824401855, -20.86729621887207, 14.341180801391602, 28.472896575927734, -22.43324851989746, 6.010183811187744, 25.045551300048828, 8.295068740844727, -14.496875762939453, 39.14017868041992, -1.8115720748901367, 24.436153411865234, -33.465972900390625, -38.6180419921875, -17.752473831176758, 0.13180220127105713, 23.645431518554688, -29.26555633544922, -22.988218307495117, 40.380226135253906, 31.14996337890625, -18.801584243774414, -8.790889739990234, -11.957160949707031, 21.30425453186035, 11.502846717834473, -34.67969512939453, 42.27547836303711, 56.959327697753906, -18.92487907409668, -21.790771484375, -31.95029067993164, -0.8583964109420776, -30.512672424316406, -36.191490173339844, -62.62662124633789, -13.645273208618164, 32.104026794433594, 27.154781341552734, -13.647706031799316, -36.669918060302734, 17.758502960205078, 14.651288986206055, -77.85194396972656, 16.867088317871094, 13.485628128051758, 18.125518798828125, -13.427875518798828, -7.385009765625, 11.345703125, 13.803013801574707, 34.91292190551758, -7.010554790496826, 8.974111557006836, -11.902688980102539, -17.17228889465332, 43.17490768432617, -31.59703254699707, -30.171142578125, 62.92543411254883, 57.326072692871094, -66.54441833496094, 74.11795043945312, 1.2878957986831665, -25.053678512573242, 9.055116653442383, 13.016345024108887, 48.63596725463867, -37.07698059082031, -14.509936332702637, 11.74940013885498, -55.349666595458984, 7.231016159057617, -82.39261627197266, -27.6570987701416, -18.800201416015625, 32.201454162597656, -5.579662799835205, -15.288050651550293, 77.6494140625, 15.633768081665039, 57.51274871826172, 1.272625207901001, 6.9003520011901855, -19.07672119140625, 1.430755376815796, 0.7794588804244995, 85.2262954711914, 9.523290634155273, -28.9797420501709, 3.7804343700408936, 0.3892834186553955, -7.620970726013184, 18.560129165649414, 3.2516794204711914, -6.21254825592041, 7.580334186553955, -31.27543830871582, -12.037532806396484, 46.2475700378418, -31.120241165161133, -15.515297889709473, -71.39138793945312, 2.7967915534973145, 29.475494384765625, -33.970035552978516, -5.693837642669678, -4.626643657684326, 36.48392105102539, 19.07107925415039, 18.592485427856445, 30.618513107299805, 44.36644744873047, 12.052407264709473, -39.64423370361328, -6.193896770477295, -1.1923842430114746, -20.943492889404297, 14.460907936096191, -89.64988708496094, -36.80320358276367, -58.02539825439453, 34.47165298461914, -1.900875449180603, -13.10265064239502, -17.37974739074707, -9.2002534866333, 12.913782119750977, -18.22185516357422, -10.650251388549805, 48.03596496582031, 16.60488510131836, 7.0891242027282715, 46.345394134521484, 14.257134437561035, -7.730984210968018, 12.145074844360352, -24.431550979614258, 18.417530059814453, -26.82034683227539, 12.490116119384766, -8.767435073852539, -35.6711311340332, -46.9673957824707, 15.712410926818848, -54.79609680175781, -22.414608001708984, -22.438644409179688, -1.8688840866088867, 23.892349243164062, 7.540427207946777, 36.980430603027344, -10.637929916381836, -38.46730041503906, 8.567697525024414, -55.02492904663086, 20.764169692993164, 56.516387939453125, -24.951229095458984, -19.614215850830078, -24.743183135986328, -39.51533126831055, 3.1947715282440186, -6.335206985473633, -37.743648529052734, -14.994135856628418, -4.7765374183654785, 8.95632266998291, -10.17882251739502, 29.93595314025879, -8.245490074157715, -14.561544418334961, 19.01148796081543, -23.075618743896484, 57.30170822143555, -80.81632232666016, 28.53193473815918, -12.63340950012207, -47.61305618286133, 5.890450477600098, 34.825469970703125, 28.103017807006836, 73.55882263183594, -37.24226760864258, -22.05782699584961, 13.325287818908691, -36.22706604003906, -35.809757232666016, -26.723411560058594, -0.0869438648223877, -17.48879051208496, -13.046584129333496, 46.59819793701172, 14.644275665283203, 9.671833038330078, 32.425254821777344, 38.845733642578125, -1.206611156463623, 3.9235897064208984, 28.134511947631836, 51.65812301635742, 27.381500244140625, -26.55858612060547, -27.652036666870117, 63.109561920166016, -3.4831457138061523, 79.38353729248047, 13.138147354125977, 41.31504440307617, 63.07853698730469, 57.08900833129883, -2.3820409774780273, -10.268559455871582, -0.5655006170272827, -19.386476516723633, 46.107139587402344, -46.05414962768555, 31.71331787109375, -16.493532180786133, -14.0248441696167, -48.010223388671875, -41.76852035522461, 12.111420631408691, 19.338943481445312, -1.2987468242645264, -6.032133102416992, 28.98868751525879, 16.943235397338867, -12.75899887084961, 32.952613830566406, 13.88068962097168, 12.634775161743164, -80.70558166503906, -13.532764434814453, -5.831214427947998, -15.128738403320312, 12.423398971557617, -2.2046451568603516, 22.11740493774414, 52.14947509765625, 8.16997241973877, -14.558040618896484, -3.2043955326080322, -13.042098045349121, 47.73060989379883, -74.2354965209961, -36.815128326416016, -5.104066848754883, 33.24542236328125, -12.212800979614258, 62.811546325683594, 53.09416580200195, 2.6422910690307617, -11.334205627441406, -44.632850646972656, -14.894708633422852, -19.69785499572754, -41.116363525390625, -37.666324615478516, 15.263633728027344, -54.53872299194336, 15.517424583435059, 9.406479835510254, 2.5445587635040283, 9.789902687072754, 74.3534164428711, -24.473365783691406, -4.355858325958252, 16.61083221435547, 15.50797176361084, -30.09581184387207, -28.459142684936523, -51.726112365722656, -14.73486042022705, -14.069269180297852, 65.1300048828125, 12.218975067138672, 16.131319046020508, 0.5552279949188232, -56.41175842285156, 1.9839508533477783, 25.028156280517578, 17.84284782409668, 17.246902465820312, 10.689277648925781, 85.43462371826172, 13.710512161254883, 11.507450103759766, -7.243607044219971, 23.91448211669922, -41.33278274536133, 25.723257064819336, 46.98310852050781, -46.69285583496094, 10.052606582641602, -15.391407012939453, -35.750492095947266, -5.545794486999512, -16.300247192382812, -42.09455490112305, 0.21322739124298096, -23.027713775634766, 3.2316691875457764, -40.50237274169922, -37.78862380981445, 44.68597412109375, 6.8447442054748535, -49.604331970214844, -13.489418029785156, 48.945343017578125, 46.04254150390625, -28.661989212036133, -17.782472610473633, -15.649249076843262, -55.04908752441406, 4.224147319793701, 10.475570678710938, 4.840761184692383, -90.93157958984375, 9.887794494628906, -23.247024536132812, -27.989377975463867, 19.45937728881836, -31.33248519897461, 44.67621612548828, -18.472627639770508, 0.735874593257904, 38.3033447265625, -18.54426383972168, 8.608242988586426, 9.45461654663086, 0.04192735254764557, 20.19779396057129, -33.81683349609375, -19.78012466430664, -8.239845275878906, 15.060735702514648, -17.989845275878906, -23.867429733276367, 43.2446403503418, -25.355234146118164, 10.369974136352539, 23.147085189819336, -20.136150360107422, 33.46400833129883, -33.915340423583984, -42.166378021240234, 23.135875701904297, -3.7243008613586426, 18.61432647705078, -3.8876075744628906, 27.00931739807129, 52.763370513916016, 5.8936767578125, -23.57425308227539, 4.849889278411865, 23.177959442138672, -3.002723217010498, 78.55064392089844, 5.661796569824219, 17.269058227539062, -21.423259735107422, -1.8183460235595703, -24.604949951171875, -45.1412467956543, 38.95859909057617, -41.48595428466797, -32.35297393798828, 8.428389549255371, -15.941305160522461, 12.885270118713379, -5.058374881744385, -39.50191879272461, -4.999364376068115, 12.481371879577637, -15.622696876525879, 0.20732903480529785, -31.162832260131836, 2.090069532394409, 36.129058837890625, -10.025713920593262, 14.042036056518555, -9.061650276184082, 14.013102531433105, -15.014787673950195, 7.541520118713379, -5.38079309463501, -1.4605003595352173, -35.171810150146484, 43.047943115234375, 32.20011520385742, 16.885770797729492, 53.89308547973633, -35.29679489135742, -7.79509162902832, 18.153409957885742, 3.4175589084625244, 23.393856048583984, -3.4597525596618652, 40.00517654418945, 32.8205680847168, 76.5653076171875, -21.98665428161621, -44.644474029541016, 32.392879486083984, 15.087989807128906, -12.506834983825684, -27.33022689819336, 8.65601921081543, 46.148277282714844, -55.60519790649414, -16.3802490234375, -17.58649253845215, 1.215823769569397, 61.240081787109375, -12.734514236450195, -16.160762786865234, 3.769502878189087, 31.872230529785156, -10.297401428222656, 35.197425842285156, 36.688724517822266, 47.334781646728516, -25.638763427734375, -21.639795303344727, -3.8391189575195312, -88.93085479736328, -50.7747802734375, -38.750877380371094, -38.8282356262207, -17.179075241088867, 45.666595458984375, -28.961200714111328, 20.08745574951172, 13.176990509033203, -9.167024612426758, 61.76604080200195, -12.543882369995117, -8.393442153930664, 25.909963607788086, 4.259829044342041, -49.37955856323242, -10.287642478942871, -15.3900785446167, -32.56993103027344, 11.165215492248535, -9.15185546875, -3.1335744857788086, 16.862197875976562, -86.05232238769531, 3.8943421840667725, -1.7335858345031738, -5.689895153045654, 35.74754333496094, 52.42191696166992, -3.6696605682373047, 25.369102478027344, -32.31325149536133, -9.014001846313477, 47.02564239501953, -15.040066719055176, -39.411956787109375, -17.713485717773438, 66.13196563720703, -4.513864994049072, -72.0868148803711, 5.386401653289795, 12.962024688720703, -17.86893653869629, 11.586066246032715, 14.25713062286377, -29.202857971191406, 75.59469604492188, 13.11251449584961, 11.860092163085938, 25.172008514404297, 16.30429458618164, -26.362489700317383, 40.694976806640625, -0.11527496576309204, 8.374954223632812, -22.328214645385742, 19.903339385986328, 35.33036804199219, 11.397432327270508, 8.173830032348633, -41.064964294433594, -16.753355026245117, 50.57243347167969, 24.067768096923828, 13.53019905090332, -42.568321228027344, -44.69994354248047, 35.9084358215332, -11.162426948547363, -24.561906814575195, 61.84645462036133, 21.246395111083984, 19.783910751342773, 26.952472686767578, 22.317337036132812, 21.89837646484375, 25.542699813842773, 18.17388153076172, 22.72060203552246, -22.517345428466797, 14.651337623596191, 46.30046081542969, -42.769554138183594, 22.260812759399414, -0.2592233419418335, 4.019747257232666, -17.05292510986328, 19.894245147705078, 19.831218719482422, 2.313953161239624, -21.8876953125, 40.9901123046875, -30.329418182373047, -11.578802108764648, -34.50328826904297, -4.198873519897461, -10.94426441192627, 1.4515016078948975, -3.1667540073394775, 0.46500664949417114, -43.099822998046875, 32.529563903808594, -32.858985900878906, 17.071956634521484, -29.477136611938477, 28.245328903198242, -3.1891262531280518, 11.962228775024414, -8.246133804321289, 35.53278350830078, 7.32283353805542, -16.066268920898438, 16.966796875, 4.050998687744141, 28.732988357543945, 10.874737739562988, 1.2275636196136475, 8.001320838928223, 42.790645599365234, -6.855996608734131, -3.335469961166382, -24.482093811035156, -39.58412170410156, -0.3904520273208618, 8.006714820861816, -10.444451332092285, -7.095137596130371, -60.50178146362305, 19.773548126220703, 34.00958251953125, -41.095088958740234, 4.038242340087891, -26.884817123413086, 8.324278831481934, -26.126171112060547, -25.603919982910156, -37.174800872802734, 45.19757843017578, -49.91064453125, -41.41390609741211, -25.0631160736084, 48.64532470703125, -17.919063568115234, 28.563617706298828, -29.623088836669922, -2.8688788414001465, -32.32949447631836, -11.312989234924316, -51.666324615478516, -1.9373806715011597, -23.331010818481445, 33.61725616455078, 27.051485061645508, -37.210872650146484, 0.4856046438217163, 32.603515625, -3.7185869216918945, 37.423133850097656, 63.476016998291016, -50.443180084228516, -17.22869110107422, -31.354774475097656, -41.150665283203125, -11.439988136291504, 17.425575256347656, 44.16548156738281, 5.306164264678955, 5.675561428070068, -26.73122787475586, 56.7717399597168, 45.18411636352539, -36.47294616699219, -11.424246788024902, 21.27191925048828, -0.9844081997871399, -51.367671966552734, 0.36410725116729736, -63.90867614746094, 4.4288201332092285, -42.31415557861328, -14.928929328918457, 6.924709320068359, -23.773584365844727, -4.137180805206299, 13.041807174682617, -9.086904525756836, -64.62084197998047, 26.76548957824707, 3.9355251789093018, 6.720259189605713, 5.7205281257629395, -36.252193450927734, -10.724074363708496, -32.787837982177734, 10.44409465789795, -9.627235412597656, 18.50372886657715, -4.991389751434326, -2.177903175354004, 10.397110939025879, 25.51384925842285, 5.2907795906066895, -38.967037200927734, 8.577027320861816, -17.77372932434082, 24.219715118408203, 20.567840576171875, 26.85589027404785, 35.85795974731445, -8.050561904907227, -26.68975830078125, -17.579734802246094, -9.506096839904785, -41.42744827270508, -27.694217681884766, -9.384137153625488, 26.159469604492188, 34.051673889160156, -14.393692016601562, 23.808876037597656, -44.42486572265625, -21.62325668334961, -24.577608108520508, -20.451698303222656, 1.2818810939788818, 87.46452331542969, -7.05939245223999, 1.7821354866027832, -41.341339111328125, 6.088820934295654, 23.783676147460938, -87.41215515136719, 29.761093139648438, 4.1804118156433105, 20.316129684448242, -34.72117614746094, -41.778079986572266, -17.72547149658203, 50.786293029785156, 2.405012845993042, -26.428667068481445, 33.403114318847656, -9.620227813720703, 7.279031276702881, 38.26435470581055, 53.76704788208008, 18.58165168762207, -53.54369354248047, -28.40852165222168, -37.69075393676758, -34.012290954589844, -11.106229782104492, -68.75492858886719, 28.88617515563965, 25.216548919677734, -12.126688003540039, 33.9861946105957, 16.021339416503906, -25.807472229003906, 26.1588134765625, -13.790528297424316, -10.316117286682129, -32.647640228271484, -10.730965614318848, -50.87623977661133, -35.126338958740234, 26.324207305908203, 24.181777954101562, -44.956451416015625, -26.02562713623047, -15.655303955078125, -74.41429901123047, -9.043581008911133, -20.905092239379883, -6.209235668182373, 3.5487208366394043, -28.336149215698242, 34.964603424072266, 12.005989074707031, -31.625268936157227, -9.389198303222656, -2.5062294006347656, -17.330307006835938, 4.453382968902588, -10.109231948852539, -17.920087814331055, 46.536712646484375, 10.237534523010254, 28.142148971557617, 35.7656364440918, 61.23246383666992, -12.125707626342773, -0.0304793119430542, 9.37153434753418, -75.02381134033203, -18.136157989501953, 8.531641960144043, -39.17304229736328, -7.6042656898498535, -53.08203887939453, -49.43600845336914, -20.65163803100586, 25.553247451782227, -0.04807627201080322, 21.781837463378906, 18.6060733795166, -23.541685104370117, 28.311498641967773, -39.46330642700195, 22.391746520996094, 68.54045104980469, -31.472900390625, -4.816857814788818, -9.984099388122559, 35.81542205810547, -81.22917175292969, -20.881633758544922, 16.831239700317383, 47.908687591552734, -26.871442794799805, 43.84235382080078, -12.877095222473145, -38.126564025878906, 9.43487548828125, -57.68923568725586, -44.195980072021484, -4.551887035369873, 15.195412635803223, -24.941591262817383, -6.432466983795166, 0.7497246861457825, 19.785573959350586, 1.4022117853164673, 38.91545104980469, -22.248249053955078, 2.1698648929595947, -21.353771209716797, -63.487464904785156, 41.933223724365234, -25.02454948425293, 46.765682220458984, -28.201217651367188, 24.67568588256836, -1.1368262767791748, -30.059402465820312, 43.19895553588867, 7.973921775817871, 33.32488250732422, -38.21360397338867, -55.100738525390625, -6.374663829803467, 26.45768928527832, -2.636103391647339, 24.395048141479492, -3.4695944786071777, 6.463295936584473, -67.70240020751953, 27.863388061523438, 33.20829391479492, 3.791775941848755, 16.119117736816406, 29.803926467895508, -40.35091781616211, -24.90730857849121, 37.99488830566406, -5.869612693786621, -13.289878845214844, 21.031286239624023, -54.84996032714844, 4.245431423187256, 28.359689712524414, -46.09408187866211, 9.145098686218262, -21.88233184814453, 6.427909851074219, 31.355228424072266, -19.068912506103516, 18.1966552734375, 2.891627788543701, -13.098228454589844, 62.94660949707031, -40.93575668334961, 29.54508399963379, 29.4616756439209, -32.349552154541016, 35.67878341674805, -17.09562873840332, 16.487031936645508, -36.19192886352539, 25.17708396911621, -40.847957611083984, -4.725303649902344, 6.9227190017700195, 18.077802658081055, 29.8571834564209, 18.46170997619629, 60.73704147338867, 14.266000747680664, 13.349092483520508, 4.166815757751465, 48.03891372680664, 63.72010803222656, 25.737462997436523, -54.12870788574219, 66.5478744506836, 74.79948425292969, 34.999549865722656, -6.833883762359619, -0.1431700885295868, 8.860037803649902, -0.5689684152603149, 25.7808837890625, 7.2651448249816895, 13.792501449584961, -6.163604736328125, -54.923946380615234, -5.79085636138916, 32.70170974731445, 22.77154541015625, 25.69606590270996, 14.103803634643555, 2.2758266925811768, -20.85543441772461, -37.5753173828125, -19.888202667236328, 17.717721939086914, 51.83071517944336, -20.838085174560547, 45.856590270996094, 19.562238693237305, -45.92932891845703, -17.76382064819336, -36.00741958618164, -6.565031051635742, -7.271644115447998, -5.965845108032227, 52.424190521240234, 53.3270378112793, 9.003174781799316, 5.158989906311035, -49.92298126220703, 1.663872241973877, -7.823512554168701, -17.76859474182129, -3.2265143394470215, 54.639305114746094, 15.835906982421875, -33.46669387817383, 8.024815559387207, 15.861820220947266, -8.700382232666016, 18.044967651367188, -7.44390869140625, -18.613922119140625, 7.271187782287598, -6.505871772766113, -18.650901794433594, -38.71062469482422, 10.362992286682129, 21.383127212524414, -29.724220275878906, -12.061939239501953, 47.952857971191406, 38.046531677246094, -3.529477834701538, 43.726959228515625, 3.060263156890869, -14.799483299255371, 34.28574752807617, -3.457493543624878, -16.692834854125977, -32.381385803222656, -22.179515838623047, 14.354104995727539, 22.108545303344727, -9.20928955078125, 1.2769969701766968, 20.80645751953125, 64.09271240234375, -73.85588073730469, -66.65875244140625, 57.005802154541016, -42.614200592041016, -59.810874938964844, -61.915245056152344, -53.228939056396484, -35.29741668701172, -5.163866996765137, 59.98634719848633, -27.798194885253906, -3.975585460662842, 6.590002536773682, -69.6362075805664, -5.530274391174316, -74.44398498535156, -26.382349014282227, 38.813785552978516, -19.530689239501953, -22.200632095336914, -6.864832878112793, -10.339223861694336, -16.877111434936523, -24.9453125, 19.567699432373047, 72.49848175048828, -26.701038360595703, 9.896913528442383, -16.768766403198242, -9.020063400268555, -14.082511901855469, 17.878835678100586, -64.45709228515625, -44.364524841308594, 14.149394035339355, 35.64740753173828, -43.94647216796875, 9.681448936462402, -1.0140243768692017, 23.227394104003906, -30.995594024658203, 42.031700134277344, 4.176971435546875, -21.85248374938965, 44.942447662353516, 49.389286041259766, -38.73025131225586, 82.64373016357422, 61.6633186340332, 47.28919982910156, 10.926486015319824, -27.61786460876465, -15.791223526000977, -26.663806915283203, -51.6547966003418, -22.6888427734375, -52.142913818359375, -49.39866256713867, 13.971856117248535, 13.183963775634766, -19.188549041748047, -12.322663307189941, -7.082894325256348, -4.753849029541016, 37.89900207519531, -11.283482551574707, -9.736404418945312, 7.6569414138793945, -34.836341857910156, 19.149738311767578, -8.804089546203613, -16.72380828857422, 0.6223217248916626, -8.62981128692627, -4.721099853515625, 17.46390724182129, -14.478671073913574, 24.215824127197266, -17.891218185424805, 33.49162292480469, 14.616190910339355, 11.95098876953125, 18.586429595947266, 40.30399703979492, -1.9049896001815796, -49.427825927734375, -1.547441005706787, 19.787704467773438, -24.376644134521484, 36.130615234375, -21.952775955200195, -8.323604583740234, -14.513107299804688, 24.95698356628418, -10.132094383239746, 68.08275604248047, -35.5985107421875, 36.85688400268555, 30.445951461791992, -29.13245391845703, -50.964805603027344, -19.204919815063477, 9.117142677307129, 28.51272201538086, -23.036081314086914, -30.428674697875977, 16.19745445251465, -17.48259925842285, -20.751968383789062, 5.351726531982422, 30.32035255432129, 34.189361572265625, -48.84455490112305, -25.094966888427734, 37.57310485839844, -26.924972534179688, 17.032072067260742, -35.75553512573242, -5.993546962738037, 40.98398208618164, 13.132147789001465, -0.5486401915550232, 6.170464992523193, 20.27355194091797, 1.3547308444976807, -4.628539562225342, -26.70861053466797, -17.656047821044922, -2.1979928016662598, 12.92117691040039, 12.520612716674805, 31.504192352294922, 5.749354839324951, -9.258155822753906, -22.88084602355957, 3.705669403076172, -51.64406967163086, 8.553400039672852, -16.690582275390625, -33.199623107910156, 31.88338851928711, 29.938613891601562, -33.63242721557617, -65.32319641113281, -30.43250274658203, -7.699638843536377, -15.563688278198242, 2.0855727195739746, 45.832210540771484, -15.933032989501953, -45.71891403198242, 44.77213668823242, 24.28814125061035, 0.8484160900115967, 16.792579650878906, 5.354912281036377, -60.781898498535156, -14.772944450378418, -3.977280616760254, 9.903448104858398, 24.552656173706055, -26.69408416748047, 8.101438522338867, 44.37065505981445, -57.59360885620117, -15.338274955749512, -2.276442766189575, -34.891929626464844, 37.000850677490234, 34.39145278930664, -15.314017295837402, 0.5246412754058838, -81.76029968261719, -26.653072357177734, -9.946249008178711, 3.677222967147827, -36.40683364868164, -68.06057739257812, -82.53207397460938, 16.052955627441406, -19.379165649414062, 1.4140474796295166, 28.287172317504883, 4.894576072692871, 12.968043327331543, -13.405265808105469, -28.304460525512695, -12.857009887695312, -5.001930236816406, -20.368806838989258, -28.09488296508789, 46.18507766723633, -38.542442321777344, 43.17584991455078, -53.00777053833008, 6.041025161743164, 17.105575561523438, -4.594385147094727, -34.70559310913086, -22.169605255126953, -7.506577014923096, 4.199440002441406, 36.685237884521484, 39.03196716308594, -23.559436798095703, 2.34124493598938, -35.163490295410156, -13.184499740600586, 26.39582633972168, 10.100547790527344, 4.8772149085998535, 47.374908447265625, -21.61478042602539, -16.600360870361328, -4.761626243591309, 7.598731994628906, 31.965946197509766, 17.453819274902344, -56.54111862182617, -5.6230058670043945, -22.702516555786133, -32.403648376464844, 42.37385940551758, -13.603811264038086, -13.803454399108887, -23.489686965942383, -52.88896179199219, 28.564096450805664, -35.775177001953125, -31.558311462402344, 14.81559944152832, -13.1709623336792, 4.034358978271484, 40.77518081665039, -32.88006591796875, 27.327133178710938, 20.577301025390625, 1.335219383239746, -15.494180679321289, -29.01877212524414, -34.143455505371094, -2.5068249702453613, 4.2534708976745605, -56.8758544921875, 34.6863899230957, 23.346900939941406, -10.517765998840332, 64.6576919555664, -1.5206730365753174, -34.14213180541992, -57.03417205810547, -13.218056678771973, -8.033346176147461, -1.5079606771469116, 37.70819091796875, 60.3174934387207, -28.140159606933594, 6.4700751304626465, -26.892301559448242, -11.443221092224121, 10.431304931640625, -13.839950561523438, -5.498312950134277, 67.33292388916016, -26.476673126220703, 4.85881233215332, -23.579273223876953, 24.95476531982422, 36.29985809326172, 4.3687663078308105, 44.51021957397461, -35.401493072509766, -28.976688385009766, -10.76810359954834, -48.739200592041016, 22.63069725036621, 6.171358108520508, 4.549166679382324, -26.8898983001709, 19.700292587280273, -13.734898567199707, -27.738296508789062, -3.891679286956787, -53.047420501708984, -16.177209854125977, 43.21660614013672, -0.630455493927002, 17.46709442138672, 27.672508239746094, -8.417922973632812, -9.446260452270508, -36.768043518066406, -16.949871063232422, 30.496614456176758, 37.00031280517578, -11.615376472473145, -12.416751861572266, -23.65361213684082, -33.6552619934082, -31.269437789916992, 16.034635543823242, -13.454428672790527, -48.210147857666016, 4.8635125160217285, -80.69990539550781, 48.231895446777344, 8.613430976867676, 46.6053352355957, -41.12889099121094, 20.231380462646484, -4.344909191131592, 38.43406677246094, 7.587642192840576, 28.126758575439453, -6.450039863586426, -44.5610466003418, 16.516998291015625, 12.483959197998047, 20.43938446044922, 60.07963943481445, -6.0606608390808105, -29.765626907348633, 7.7960205078125, 10.471506118774414, -2.6766767501831055, 16.896282196044922, 62.31875228881836, 38.64679718017578, 37.19817352294922, 5.07276725769043, -44.958457946777344, 29.86164665222168, -27.679962158203125, 51.269004821777344, 51.49787902832031, -62.27814865112305, 41.42629623413086, 22.102304458618164, -7.237597942352295, 15.201656341552734, 41.9284782409668, -21.8430233001709, -27.468975067138672, -42.04997634887695, 39.07181167602539, -60.184661865234375, -24.730323791503906, -42.525123596191406, -20.212846755981445, -24.87186050415039, 5.556357383728027, -16.144704818725586, 81.54470825195312, 1.3012202978134155, 12.351759910583496, -17.057058334350586, -25.834522247314453, -16.24822235107422, 1.0067644119262695, -0.5000319480895996, -21.683780670166016, 6.947997093200684, -16.836383819580078, -9.373833656311035, 18.93983268737793, 22.81719970703125, -23.94036293029785, 8.111242294311523, 2.014434576034546, -38.82191467285156, -14.222307205200195, 31.350255966186523, -15.974065780639648, -25.165245056152344, -22.255115509033203, -40.575584411621094, -24.941930770874023, 32.91635513305664, -17.694276809692383, 20.504547119140625, -20.790077209472656, -37.84904479980469, -9.491010665893555, 49.2074089050293, -13.497194290161133, -2.042720317840576, 41.36447525024414, -44.71849822998047, -14.905458450317383, 16.602489471435547, -6.651721954345703, 16.494590759277344, -6.718523979187012, -40.45524978637695, -23.299015045166016, -58.662742614746094, 37.93522644042969, -29.219383239746094, -8.638566017150879, -16.051828384399414, -40.602333068847656, 12.642821311950684, 55.48440170288086, -6.543508529663086, -70.65730285644531, -62.844669342041016, 5.989832401275635, 6.865355968475342, -4.405086517333984, -49.97977828979492, -38.535308837890625, 50.1107063293457, 32.64752197265625, -47.36238479614258, -37.04024124145508, 19.717309951782227, 28.67653465270996, -5.171869277954102, 8.252299308776855, 15.976336479187012, 0.23655924201011658, 4.334075927734375, -49.0346565246582, -2.1925160884857178, -49.992469787597656, 29.386882781982422, -37.158817291259766, 9.749502182006836, -1.495201826095581, -9.307941436767578, 38.41915512084961, -7.7048234939575195, -30.705799102783203, -34.1268310546875, 20.195280075073242, 13.48757266998291, -23.01786994934082, 47.15388107299805, -6.442544460296631, -7.851060390472412, -11.498079299926758, -11.09083366394043, -12.127580642700195, -42.293060302734375, -51.21113586425781, -20.16394805908203, -7.9484992027282715, 6.173561096191406, -15.230864524841309, 19.690174102783203, 14.332297325134277, -38.577064514160156, -7.258494853973389, -8.475251197814941, 0.7871682047843933, -9.450904846191406, -23.845563888549805, 11.856189727783203, -46.33275604248047, 98.462890625, -36.87079620361328, 11.86966609954834, 20.519329071044922, -4.601630210876465, -50.883541107177734, 35.140533447265625, -22.243684768676758, 8.067931175231934, 14.571495056152344, -32.69425582885742, -24.491132736206055, 24.844738006591797, 41.35795974731445, -23.361997604370117, -7.680843830108643, 38.86749267578125, 1.0665779113769531, 80.5077133178711, 22.603403091430664, 31.876012802124023, -25.81583595275879, -26.321144104003906, 5.369273662567139, 18.477031707763672, -33.88682174682617, -5.2189836502075195, -31.32871437072754, -2.3580243587493896, -4.029278755187988, 7.381119728088379, -4.946783542633057, -19.26170539855957, -29.49100685119629, 38.4445686340332, -2.384614944458008, 9.428675651550293, 48.52859115600586, 1.5607733726501465, -37.64925003051758, 8.163642883300781, -31.86060905456543, 18.17605972290039, -16.304655075073242, -13.034960746765137, 11.792750358581543, -28.29974365234375, -72.2037124633789, -63.38594436645508, 13.122587203979492, 16.52338409423828, 20.145429611206055, 7.577631950378418, 25.854568481445312, 43.47244644165039, -32.544586181640625, -24.22187614440918, 37.53419494628906, 18.75471305847168, -0.5312697887420654, 15.923690795898438, 12.256731986999512, -20.999685287475586, -13.5927152633667, 28.632381439208984, 12.269572257995605, -27.21857261657715, 13.137751579284668, 1.5973482131958008, -26.71129608154297, -36.81031036376953, 11.613485336303711, 21.345563888549805, -9.672986030578613, -26.827190399169922, 30.008602142333984, 27.401329040527344, -8.438278198242188, -17.483728408813477, -2.5502266883850098, -30.662660598754883, -0.2892458438873291, -25.34600067138672, -19.940757751464844, -36.05742645263672, -39.30342102050781, -31.70602035522461, 39.715274810791016, -31.687740325927734, 39.760128021240234, -31.48611068725586, -42.00278091430664, 3.106992721557617, 19.559175491333008, 18.00740623474121, 24.776952743530273, -9.959383010864258, -29.150957107543945, -31.425012588500977, -62.78192138671875, 53.604793548583984, -27.594377517700195, 67.42070007324219, -20.530319213867188, -43.84123611450195, 4.773509979248047, -33.91543960571289, -48.58236312866211, 26.009763717651367, -9.899330139160156, -26.104633331298828, -27.226490020751953, -18.10054588317871, 11.49907112121582, 15.09259033203125, -14.621981620788574, -22.387615203857422, -19.2719669342041, -9.141032218933105, 49.470516204833984, 41.9794921875, 10.40342903137207, 19.755735397338867, 38.72782516479492, -40.09815216064453, -30.214168548583984, -15.77733325958252, 21.389177322387695, 12.623162269592285, 33.171592712402344, -39.11585998535156, 19.420833587646484, -42.07405090332031, 33.733428955078125, -6.929326057434082, 6.008519649505615, 11.40695571899414, -41.67747497558594, -14.709864616394043, 21.298677444458008, -31.900117874145508, -26.985610961914062, 19.193193435668945, -13.052030563354492, -9.092076301574707, 0.05631875991821289, 13.425087928771973, 0.40354660153388977, 56.02641296386719, 27.083240509033203, -8.153437614440918, -35.075599670410156, 14.405344009399414, -0.4474155902862549, -42.524497985839844, 13.241353034973145, 14.432252883911133, -8.719237327575684, 50.46663284301758, -22.927886962890625, 43.858978271484375, -3.6499154567718506, -2.3665354251861572, -16.279558181762695, 21.440797805786133, 3.630455732345581, 0.7072230577468872, -17.486934661865234, 1.0681657791137695, -2.616994857788086, 27.889009475708008, 5.014486312866211, 3.9380340576171875, -8.509663581848145, -6.781693935394287, 12.597052574157715, 21.451290130615234, -14.894210815429688, 7.836639404296875, 21.49436378479004, 23.0203800201416, 4.10914945602417, -22.81532096862793, 15.743840217590332, -1.6374506950378418, 34.465919494628906, 4.213497638702393, 11.897244453430176, 29.099231719970703, -50.263458251953125, 31.581565856933594, 20.391559600830078, 8.31694507598877, 80.2196044921875, 25.67159652709961, -39.831974029541016, -1.0637006759643555, -6.801140308380127, -22.12126922607422, -17.78809928894043, 35.923954010009766, 0.669447660446167, 24.8295955657959, -34.149330139160156, -6.223271369934082, -20.622350692749023, 3.651704788208008, -17.520654678344727, -7.876539707183838, 11.435386657714844, 22.812925338745117, -43.57688522338867, -0.11530381441116333, 19.53831672668457, 0.48191577196121216, 49.37082290649414, -17.664365768432617, -41.77300262451172, -36.439117431640625, 33.07401657104492, 7.646521091461182, -33.299774169921875, 38.353904724121094, -1.4201449155807495, 11.403488159179688, 69.09097290039062, 28.150310516357422, 59.82229995727539, 11.390983581542969, 60.240726470947266, -39.211669921875, -49.184757232666016, 21.56826400756836, -30.720458984375, -23.179548263549805, -57.7217903137207, -13.051196098327637, 42.728023529052734, -12.032790184020996, 43.19677734375, -26.84766960144043, -52.67560958862305, -45.69271469116211, -12.863119125366211, -2.676635265350342, 19.239225387573242, 28.78377342224121, -3.720529079437256, 29.358356475830078, -3.9763572216033936, 9.747065544128418, -27.114818572998047, 15.63267707824707, 27.880294799804688, -17.64809226989746, 18.70626449584961, -9.04025936126709, 59.78477096557617, 1.9682706594467163, 38.33865737915039, 7.103856086730957, 12.65680980682373, 36.61215591430664, 11.655394554138184, 39.32551574707031, 3.062788963317871, 22.364429473876953, -20.760061264038086, -8.334861755371094, -0.260934442281723, -13.192577362060547, 32.175052642822266, -3.0385513305664062, 21.89500617980957, 58.252315521240234, 22.970359802246094, 1.0944254398345947, 9.014028549194336, 47.13444900512695, -4.983529090881348, 41.61851119995117, 46.60003662109375, -34.87007522583008, 19.00313377380371, 14.035174369812012, 17.499988555908203, -32.23635482788086, 5.237541198730469, -13.396153450012207, 6.746856689453125, 37.307830810546875, -21.224369049072266, -30.331640243530273, 58.82022476196289, -46.819393157958984, -13.663249969482422, 39.150936126708984, 21.948108673095703, 29.901824951171875, -5.6036481857299805, 36.123661041259766, -72.08685302734375, -0.506535530090332, -13.554864883422852, -8.152238845825195, -12.129183769226074, -15.896696090698242, -53.17745590209961, -18.098896026611328, 35.18904113769531, -20.697416305541992, 12.552711486816406, -35.83314895629883, -7.567492961883545, 26.991003036499023, 24.526168823242188, -5.651188373565674, 9.310897827148438, 6.8951191902160645, -24.17393684387207, 26.23009490966797, 7.08364725112915, 25.516002655029297, 18.063636779785156, 50.658294677734375, 14.152963638305664, -8.711970329284668, 16.15613555908203, 16.642736434936523, 4.550179481506348, -8.688774108886719, -15.643365859985352, 21.361738204956055, -18.557167053222656, -16.633211135864258, -20.211137771606445, -33.21604919433594, -11.165064811706543, 40.22467041015625, -3.0904040336608887, -17.129613876342773, -16.224098205566406, -50.247947692871094, -16.064775466918945, 4.063449382781982, -20.732460021972656, 9.438237190246582, 28.487472534179688, 15.30441951751709, -20.398235321044922, -18.561450958251953, 4.712864875793457, 30.7556095123291, -61.42904281616211, -39.2254524230957, 38.2559700012207, 5.285528659820557, -23.56214141845703, -25.658903121948242, -26.04075813293457, 26.63920021057129, 30.08639144897461, 0.37315258383750916, 50.10942840576172, -19.138561248779297, -9.416841506958008, -36.7554817199707, -0.9802004098892212, 14.652647972106934, 56.00067901611328, 79.5633544921875, 17.59157943725586, -5.773487091064453, -19.4484920501709, -0.31468936800956726, 7.506411075592041, 23.057384490966797, 6.714874744415283, 17.58806800842285, 8.985310554504395, 3.5916588306427, 29.473196029663086, -6.58917236328125, -29.681243896484375, -22.003889083862305, -16.094314575195312, 71.29903411865234, -34.66537094116211, -17.326391220092773, -17.692201614379883, 60.24339294433594, 13.833833694458008, 32.95575714111328, -47.619789123535156, -7.522809982299805, 15.552412986755371, -8.159222602844238, 30.317493438720703, 54.79262924194336, -32.29137420654297, -27.0593204498291, 35.4815673828125, 37.836483001708984, 25.86210823059082, 38.27344512939453, -37.562904357910156, 16.35932731628418, -0.08662059903144836, 15.464056015014648, 11.023397445678711, 74.72239685058594, 61.673561096191406, 5.2963480949401855, 46.92363739013672, 33.16483688354492, 24.181493759155273, 47.71818542480469, 54.593299865722656, -16.069908142089844, -21.448278427124023, -26.046985626220703, -14.370834350585938, -8.642495155334473, -0.2464696168899536, 6.029672622680664, -14.323027610778809, 7.140607833862305, 16.01189422607422, 38.94843292236328, 33.25522232055664, 3.140946626663208, -24.95917510986328, 71.4411849975586, -18.457698822021484, 6.857615947723389, -3.1342737674713135, -10.069366455078125, -21.583168029785156, 22.013999938964844, -51.63142395019531, 41.51687240600586, 18.595672607421875, -28.90289878845215, -4.625810623168945, 6.223403453826904, 3.0894174575805664, 77.61036682128906, 32.964744567871094, 28.790185928344727, -9.810319900512695, 25.03961181640625, 2.38895845413208, 42.72637939453125, 11.488162994384766, 5.23900842666626, 21.739797592163086, 65.91040802001953, 24.111478805541992, -12.181231498718262, 40.679988861083984, -25.764236450195312, 47.35248565673828, 57.92787170410156, -14.845479011535645, 19.471376419067383, -35.33147048950195, 45.867610931396484, -11.611495971679688, -13.670809745788574, 10.071020126342773, 49.18890380859375, -32.07878112792969, -10.252718925476074, 42.02263259887695, -27.81068229675293, -8.100617408752441, -31.537912368774414, -13.362637519836426, 71.53539276123047, 55.93250274658203, 3.7289628982543945, -3.1543052196502686, 56.380924224853516, -10.620124816894531, 24.788087844848633, 15.289153099060059, 4.996878147125244, 10.764158248901367, 54.09530258178711, -16.16872787475586, 39.11396026611328, 6.768303871154785, -44.01043701171875, 4.694805145263672, -12.550798416137695, 36.365455627441406, 28.640981674194336, -36.15211868286133, -29.34852409362793, 15.55478286743164, 39.35636520385742, -6.3503594398498535, 6.073554515838623, -3.333057403564453, 8.477267265319824, -4.328718185424805, -8.109752655029297, -68.96455383300781, -9.84539794921875, 13.370396614074707, -63.82369613647461, -0.35353994369506836, -13.629530906677246, 7.995030403137207, 9.05763053894043, 32.152889251708984, 17.410987854003906, -5.991506099700928, 31.362646102905273, -2.7772350311279297, 40.04437255859375, -25.816862106323242, -21.909624099731445, -39.75608825683594, -23.57996368408203, -40.61778259277344, 21.004180908203125, 0.014648795127868652, -48.1312141418457, 30.0794734954834, 28.364286422729492, 27.092811584472656, -37.470481872558594, -25.263294219970703, -56.873802185058594, -19.46320152282715, -14.529191017150879, 22.512514114379883, 51.39501953125, 31.437986373901367, -9.331740379333496, -4.531643867492676, -47.88825988769531, -11.741374015808105, 30.631019592285156, -21.100692749023438, 40.3498420715332, 6.8715667724609375, 62.01459503173828, 6.468823432922363, -14.298654556274414, -26.268789291381836, -8.813482284545898, -34.55743408203125, -48.316829681396484, -16.801855087280273, 3.5044538974761963, -9.952372550964355, -7.673063278198242, 29.922977447509766, -17.907514572143555, -1.7595272064208984, -42.98997116088867, -8.039079666137695, -29.220623016357422, -35.19491958618164, -15.22744083404541, 2.0522050857543945, -34.26206970214844, 21.211177825927734, -0.4916931986808777, 17.99493408203125, 15.164671897888184, -64.84406280517578, 43.89667892456055, 39.43808364868164, -61.72748947143555, 86.65597534179688, -1.3672115802764893, -15.659695625305176, -40.42269515991211, -11.978423118591309, 6.971639633178711, 40.67121887207031, -4.103363037109375, 22.27855110168457, -0.18144774436950684, -63.29033660888672, -58.08283233642578, -0.07833707332611084, 16.59209442138672, 20.78380012512207, 27.53978157043457, -22.081125259399414, 23.430545806884766, -3.1085994243621826, -2.954423427581787, 28.46129608154297, 14.225797653198242, -2.762969970703125, 33.308937072753906, 23.51373291015625, 19.683055877685547, 8.187287330627441, -31.701557159423828, 37.36711502075195, -16.829071044921875, -16.388221740722656, 21.859586715698242, 22.08053207397461, 42.267269134521484, 0.9170566201210022, -6.912216663360596, 18.77811622619629, 41.2457389831543, -17.143972396850586, 4.666825294494629, 38.29225540161133, -2.815464496612549, -32.67215347290039, -28.84528923034668, 15.633444786071777, -36.70246124267578, 3.9838743209838867, -7.155431747436523, 14.625555038452148, 8.99788761138916, -12.661945343017578, -3.0738141536712646, -37.649417877197266, -11.421686172485352, 56.14818572998047, 1.478055477142334, 58.824954986572266, 30.650585174560547, 54.69582748413086, -69.26923370361328, 44.94154357910156, 23.378158569335938, -34.626712799072266, 29.42940902709961, -12.15008544921875, 64.66512298583984, 6.5929999351501465, -3.565108299255371, 5.298955917358398, 4.504354476928711, 6.8605637550354, -15.051888465881348, 1.5223000049591064, 33.559356689453125, 9.509246826171875, 25.803041458129883, 11.746540069580078, -15.799333572387695, -67.64297485351562, -17.87226104736328, -22.767921447753906, -13.683110237121582, 7.010751724243164, 58.0601806640625, 9.291001319885254, 49.19336700439453, -27.808900833129883, -3.75233793258667, 12.22611141204834, 49.29689025878906, 3.4758358001708984, 12.451845169067383, 13.432738304138184, 1.7993924617767334, 3.122967004776001, -5.467643737792969, -30.866025924682617, -5.112278938293457, 0.42377275228500366, 31.418148040771484, -37.02400588989258, 21.101348876953125, 16.610675811767578, -27.047950744628906, 28.323257446289062, 1.7104029655456543, 15.802815437316895, -27.99633026123047, -22.69358253479004, -9.451356887817383, 41.689720153808594, -32.21160888671875, -0.6764177083969116, 9.31612777709961, 58.76016616821289, -16.093618392944336, -32.92625045776367, -68.89382934570312, -1.9074609279632568, -45.776039123535156, -17.894580841064453, -26.820552825927734, -7.599270820617676, 13.197936058044434, 14.42525863647461, -17.691133499145508, -3.0152742862701416, 17.85097885131836, -57.222434997558594, 29.760343551635742, -49.38103103637695, 0.31573158502578735, 5.583743572235107, -20.37249183654785, 23.72519874572754, 9.488805770874023, -10.262858390808105, -47.79825973510742, 16.146453857421875, -8.993759155273438, -5.704089641571045, 7.162511825561523, -53.785423278808594, -15.190909385681152, 16.956558227539062, -26.43082618713379, 33.04957962036133, 17.543825149536133, 13.571625709533691, 6.903618812561035, -1.310152530670166, -10.749885559082031, 32.506004333496094, 7.100116729736328, 35.20841598510742, -53.35139083862305, -21.370952606201172, 57.86545181274414, -20.846879959106445, 4.250245094299316, 38.752601623535156, 78.81554412841797, -11.447559356689453, 29.933284759521484, -5.74696159362793, -2.9474635124206543, -11.124808311462402, 44.44253921508789, -28.92384910583496, -76.31159210205078, 12.361562728881836, -3.186279058456421, -9.818848609924316, -31.467275619506836, 9.602785110473633, 2.171809196472168, 3.274528741836548, -1.4890778064727783, 63.906524658203125, 18.552480697631836, 59.1980094909668, 0.908031702041626, -9.611106872558594, -14.853277206420898, 14.441106796264648, 43.21611022949219, 5.893704891204834, 14.978297233581543, -52.184730529785156, 19.098905563354492, -0.6043073534965515, 16.460161209106445, 19.262048721313477, -4.2537994384765625, 3.4784436225891113, 19.928159713745117, -24.28962516784668, 24.11575698852539, -7.612603664398193, -10.507814407348633, 6.457371234893799, 31.142627716064453, -23.672670364379883, -9.193899154663086, 10.238946914672852, -13.15995979309082, 48.88449478149414, 41.121273040771484, 12.429079055786133, 7.4467010498046875, 62.95288848876953, 37.42207717895508, -7.195029258728027, 14.72685432434082, 24.768522262573242, -23.68817901611328, -5.278494358062744, -8.839997291564941, -35.7452392578125, 6.625066757202148, -64.680419921875, 5.228822708129883, -11.592578887939453, -25.60832405090332, 12.606752395629883, -38.454959869384766, -7.5647454261779785, -1.3298208713531494, 46.4742546081543, -16.774017333984375, 15.745077133178711, 56.672088623046875, 29.690040588378906, 16.153888702392578, 8.425593376159668, -21.673189163208008, -53.15713119506836, 21.326797485351562, -55.446922302246094, 8.016268730163574, -7.169199466705322, 24.162700653076172, -32.59312057495117, -39.76947784423828, 21.78896713256836, -8.974675178527832, 38.631752014160156, 1.1504613161087036, -23.24207305908203, 50.681732177734375, -4.407627582550049, -7.9206862449646, 28.176591873168945, -40.2690315246582, -18.80022430419922, -26.21259117126465, -6.0509748458862305, -35.71725082397461, -16.659162521362305, -29.06747055053711, 16.27155876159668, 12.91884708404541, -27.980125427246094, 14.89792251586914, 16.707468032836914, -23.881174087524414, 3.489417552947998, 6.5547003746032715, -14.126311302185059, 2.2608017921447754, -52.23284149169922, -51.96268081665039, 21.853498458862305, -31.01304054260254, 12.79500961303711, 31.21373176574707, 8.179598808288574, -11.16894817352295, -59.19721221923828, -17.371957778930664, -2.5788912773132324, -24.371917724609375, 12.888486862182617, 5.427240371704102, -18.28351402282715, -17.008670806884766, 0.0668908953666687, -2.524468421936035, -14.025054931640625, -19.788042068481445, 35.51688766479492, 46.60071563720703, -5.675084114074707, 20.124874114990234, -3.2730019092559814, 8.53139591217041, 41.94905471801758, 7.51898717880249, 33.517269134521484, 8.448508262634277, -42.77170181274414, -19.49968719482422, -46.13326644897461, -25.48352813720703, 25.861841201782227, -37.82461166381836, -14.905540466308594, 19.959842681884766, 24.175832748413086, 77.44862365722656, 4.747752666473389, -0.5361024141311646, -6.930682182312012, -19.054462432861328, 3.716320276260376, 39.17914962768555, -54.350547790527344, 5.700242519378662, 45.03256607055664, 25.584674835205078, 54.29262924194336, -11.956402778625488, 37.52604675292969, -34.50225067138672, 39.87887191772461, -32.074337005615234, 0.04381901025772095, 16.00863265991211, 21.61842918395996, 8.832501411437988, 33.33542251586914, -1.1044307947158813, -30.434572219848633, -55.623966217041016, 31.528522491455078, -6.47385835647583, -27.85394287109375, -2.9046196937561035, 54.12334442138672, -7.418406009674072, 8.164322853088379, 27.333227157592773, -40.75624465942383, -32.68234634399414, -12.478561401367188, -15.6202392578125, -28.987947463989258, 8.585049629211426, -0.5259658098220825, 32.79723358154297, -33.35114669799805, 36.15612030029297, 4.5028076171875, 32.60813522338867, 33.80816650390625, 19.68887710571289, 45.85160827636719, -49.60361099243164, -26.920974731445312, -7.202796459197998, -6.549973011016846, 11.05601692199707, 30.67473602294922, -1.246917963027954, 38.50537109375, -67.93800354003906, 3.4596176147460938, 0.4000495672225952, 37.41506576538086, -33.67416763305664, -3.3542399406433105, -7.288641452789307, 18.347318649291992, -14.739158630371094, -12.715226173400879, 20.635215759277344, -24.115219116210938, -6.416304588317871, 12.401729583740234, -12.147154808044434, 12.4787015914917, 57.94612503051758, -22.09937286376953, -29.836271286010742, -5.381415843963623, 19.186368942260742, 46.117774963378906, -1.1764212846755981, 18.523622512817383, -8.681193351745605, 7.231569290161133, 24.023286819458008, -20.71125602722168, -30.592998504638672, 42.70457077026367, 39.37370300292969, 23.805809020996094, -4.154626846313477, -54.108768463134766, -44.2185173034668, 9.552457809448242, 8.183595657348633, -66.69190979003906, 11.745444297790527, 28.67644691467285, 2.980609178543091, 26.364299774169922, -54.73345947265625, 31.56475067138672, 25.97939682006836, -5.938076019287109, 70.6272964477539, -7.876922607421875, -42.9547233581543, 10.898181915283203, -23.21907615661621, -15.164270401000977, -37.85676956176758, -34.7720832824707, 36.973548889160156, -8.358842849731445, 24.300697326660156, -30.1149845123291, -10.010421752929688, -13.288647651672363, -21.072036743164062, -1.7344402074813843, -3.071751356124878, -18.89954376220703, 2.936197280883789, -4.114075660705566, -75.2275161743164, 42.90852737426758, 70.99163818359375, 13.837128639221191, 3.6440351009368896, 61.809879302978516, 13.286500930786133, 9.246315956115723, 14.665576934814453, -20.841888427734375, 11.716001510620117, 12.603221893310547, -35.829627990722656, -8.56421947479248, 55.681026458740234, 10.665070533752441, 4.980682373046875, -19.345922470092773, 55.98529815673828, -4.832002639770508, -14.928786277770996, 14.030439376831055, 49.715423583984375, -10.567537307739258, -7.494294166564941, 12.490035057067871, 39.58769607543945, -25.357023239135742, 21.34169578552246, 25.261131286621094, -23.903339385986328, 12.516121864318848, 2.8367767333984375, 20.14668083190918, -31.976280212402344, -8.897964477539062, 1.5727128982543945, 41.90231704711914, 3.477243423461914, 11.876657485961914, 12.117713928222656, -1.6590687036514282, -2.726097345352173, 26.829505920410156, -7.839353084564209, 74.57437896728516, 0.607168436050415, -17.48008918762207, 4.385210037231445, -21.026588439941406, -33.84031295776367, -9.476011276245117, 2.721867322921753, 1.2949680089950562, 83.2147216796875, -2.2974019050598145, -20.203893661499023, 78.45428466796875, -10.039078712463379, -47.69429397583008, -29.778640747070312, 38.130943298339844, -4.071251392364502, 41.71080017089844, -14.951152801513672, -6.350513935089111, -9.094539642333984, -10.390129089355469, -19.937456130981445, 52.07826232910156, 32.92889404296875, 24.677791595458984, 33.972412109375, 9.18991756439209, -6.234096527099609, 46.46537780761719, 32.173248291015625, -6.567153453826904, 16.52545166015625, 35.12499237060547, 3.3098669052124023, 77.03004455566406, 6.092278003692627, -22.432546615600586, 3.3273215293884277, 3.569556474685669, -1.9315118789672852, 8.143501281738281, 34.52765655517578, 14.840286254882812, 4.673656463623047, 2.3371710777282715, 25.1579532623291, 38.07427215576172, -2.1154727935791016, -26.368194580078125, -5.046565532684326, 36.56570053100586, -50.041282653808594, -21.18559455871582, -24.203794479370117, -34.948001861572266, -41.12465286254883, -24.477359771728516, 13.661707878112793, -25.557903289794922, 33.68898391723633, 13.668357849121094, 40.644344329833984, 25.441959381103516, 8.932546615600586, 26.818370819091797, 54.507171630859375, -43.537254333496094, 14.063711166381836, -50.49671173095703, 42.826499938964844, 12.32201862335205, -4.592683792114258, 46.71981430053711, 22.9805965423584, 35.55332946777344, 27.503225326538086, -9.228022575378418, -15.066353797912598, -11.740011215209961, 44.43358612060547, 6.4945173263549805, 39.30377960205078, -69.17302703857422, 7.394585609436035, -13.876852035522461, -19.119583129882812, 11.484990119934082, 22.8843936920166, -64.84536743164062, -4.754042148590088, -21.088298797607422, -46.861934661865234, 38.452659606933594, 22.748964309692383, 18.154617309570312, -66.6881103515625, -38.69035720825195, -8.225800514221191, 34.60618209838867, -20.839303970336914, -29.843809127807617, -29.729516983032227, 55.673973083496094, 11.815479278564453, -19.601282119750977, 31.25902557373047, 37.910736083984375, -25.291120529174805, -51.57924270629883, -2.848932981491089, -69.5074462890625, -49.91992950439453, 7.537709712982178, 1.8976168632507324, 22.14484405517578, -23.41391372680664, -23.16374397277832, 22.119491577148438, -77.27442932128906, 24.004947662353516, 4.9965949058532715, -11.150958061218262, 11.480721473693848, 12.522165298461914, 11.217811584472656, 20.447269439697266, 39.09507369995117, -55.52412414550781, -13.060220718383789, 19.610858917236328, 4.1562042236328125, -20.808921813964844, -10.62575912475586, -10.030593872070312, -1.3782254457473755, -42.7849006652832, 27.08948516845703, 43.001686096191406, -1.3561053276062012, -1.2497508525848389, -28.69918441772461, -47.95936965942383, 7.190537929534912, 12.381050109863281, 11.971129417419434, 5.476771354675293, -11.802567481994629, -33.28109359741211, 2.9542417526245117, 22.308446884155273, 33.19641876220703, -51.084720611572266, -20.909791946411133, 22.48589324951172, -34.622039794921875, -32.1006965637207, -16.6267147064209, -22.459972381591797, 7.045037746429443, -19.011028289794922, 54.46393966674805, 8.717987060546875, 25.928495407104492, 27.551603317260742, -0.27276062965393066, -14.814672470092773, 19.89151382446289, -24.602848052978516, -31.046253204345703, -2.8090882301330566, 31.41967010498047, 13.76828384399414, 13.535126686096191, -86.81356048583984, -32.68830490112305, -19.7233943939209, -19.356138229370117, -9.734408378601074, 45.048492431640625, 11.803573608398438, -2.508662700653076, -10.61380386352539, -20.730262756347656, -29.276369094848633, -4.395057201385498, -9.337368965148926, 15.521232604980469, 36.2454948425293, 7.762939929962158, -65.9570083618164, 5.332289218902588, -33.11618423461914, -12.502076148986816, -24.266260147094727, -29.56693458557129, -11.658936500549316, 9.596639633178711, -12.809415817260742, -21.89584732055664, 22.077478408813477, 35.904441833496094, 39.5088996887207, 52.63621139526367, -3.121699810028076, 49.34406661987305, -35.67066955566406, 1.366420865058899, 59.204315185546875, -28.3778133392334, -12.981449127197266, -16.383102416992188, 7.078528881072998, -19.516494750976562, -15.628600120544434, -35.91450500488281, -55.179931640625, 2.5506014823913574, 7.890722274780273, -23.491994857788086, -21.055116653442383, 20.62788200378418, 23.677722930908203, -30.990264892578125, -36.91313171386719, -18.613344192504883, 58.28927230834961, -22.8493595123291, 53.9581298828125, -32.026954650878906], [-33.72198486328125, -7.333057880401611, 3.8385558128356934, 3.793440341949463, -24.532663345336914, 2.3431320190429688, -23.9415340423584, -22.72287940979004, 29.789966583251953, -68.27315521240234, 20.214153289794922, -21.05367660522461, -28.47745132446289, 22.17623519897461, 13.078652381896973, 16.227540969848633, 19.143945693969727, -14.443458557128906, 31.317060470581055, 11.81386661529541, -2.166731834411621, 9.628389358520508, 16.218364715576172, 43.61977767944336, 36.22756576538086, -23.66877555847168, 22.938024520874023, -2.2605814933776855, 5.908209800720215, 7.928581237792969, -27.751840591430664, 26.169681549072266, -54.7999267578125, -4.3814377784729, 0.10645055770874023, -58.22317886352539, 18.262662887573242, 10.133622169494629, -12.788816452026367, -27.719970703125, 17.890411376953125, 17.96883773803711, 13.663189888000488, 37.28561782836914, 2.5417909622192383, 24.095539093017578, -11.172327041625977, 4.8876752853393555, 49.19721603393555, 41.57643127441406, 45.774845123291016, -0.8325344324111938, 30.23548698425293, -2.735934257507324, 82.98670196533203, -29.503366470336914, -53.27503204345703, -26.240203857421875, -57.67715835571289, -63.35466003417969, -22.94713592529297, 2.1985690593719482, 53.753177642822266, -42.4991455078125, 13.954737663269043, -13.553963661193848, -11.757779121398926, 7.070924758911133, -23.000673294067383, -41.87439727783203, -19.978931427001953, -17.795204162597656, -0.7681064605712891, 46.84016036987305, -16.879070281982422, -49.14010238647461, 0.9284166097640991, -37.88895034790039, -27.134798049926758, -75.87997436523438, -16.717344284057617, 17.687868118286133, -55.783226013183594, 66.51150512695312, -24.663143157958984, 63.61789321899414, -28.861663818359375, 7.831133842468262, -19.806673049926758, 29.14596939086914, 11.057458877563477, 37.57776641845703, 4.993521213531494, -5.485050201416016, -2.9058656692504883, 44.976375579833984, 3.3518593311309814, 19.424827575683594, 7.734313011169434, -0.4977061152458191, -11.816703796386719, 39.9169807434082, -51.58574295043945, -18.462617874145508, 52.07275390625, 10.895991325378418, -16.66067886352539, 11.580160140991211, -40.279335021972656, -31.780683517456055, 18.66396141052246, -39.47092056274414, 24.974401473999023, 45.834320068359375, -6.550551414489746, -30.413204193115234, -58.10034942626953, 43.243797302246094, 23.607528686523438, -35.95057678222656, 16.45615005493164, -32.62070083618164, 54.0321044921875, 32.79104995727539, 12.069814682006836, 1.9804145097732544, -32.65987777709961, -12.43023681640625, 11.915699005126953, 1.597639799118042, 9.528903007507324, -36.9385986328125, -13.817540168762207, 2.8432631492614746, 18.77906608581543, -45.15695571899414, 22.853744506835938, -47.28376388549805, 57.01515579223633, -15.102970123291016, 37.75166320800781, 5.669926166534424, 39.948307037353516, 24.925302505493164, -42.5883674621582, -41.17144012451172, 16.210548400878906, 21.007383346557617, 15.630033493041992, -1.74260413646698, 27.594058990478516, 15.224018096923828, 25.063154220581055, -4.915171146392822, 18.31653594970703, -44.908206939697266, 6.481363296508789, -35.38825988769531, 1.563516616821289, -19.556276321411133, -14.219184875488281, -31.020036697387695, -5.457648277282715, -19.44287872314453, 26.281572341918945, 40.998538970947266, 7.458303451538086, -23.259994506835938, 22.62666893005371, 41.09716033935547, 35.74761962890625, -26.465190887451172, -19.381929397583008, 19.01470375061035, -3.3059935569763184, 9.895994186401367, -21.768272399902344, -5.694284439086914, -18.328472137451172, 7.499810218811035, -33.19750213623047, -55.93119430541992, 46.16999435424805, -65.89120483398438, -18.046859741210938, 54.74921798706055, 12.934858322143555, 53.240299224853516, -22.538225173950195, 4.279971122741699, -20.285879135131836, 16.680173873901367, 44.511314392089844, -26.555686950683594, -32.66177749633789, -17.55485725402832, -15.975932121276855, -2.5564894676208496, -8.671869277954102, -39.28300857543945, 41.46122360229492, -4.087310314178467, -57.487396240234375, -30.953372955322266, -12.34853744506836, 37.33344650268555, 6.070388317108154, -55.94697570800781, 10.626605987548828, -12.779109001159668, 64.58560180664062, -37.606048583984375, -33.73455047607422, 23.528493881225586, 80.0062255859375, 43.45664596557617, 12.215559005737305, -26.257978439331055, -55.48832702636719, 8.393220901489258, 1.3028477430343628, 42.16868209838867, 32.59241485595703, 5.283843994140625, -49.929927825927734, -20.825748443603516, -6.531705856323242, -7.447615623474121, 60.77070999145508, 8.13176441192627, -80.46115112304688, 5.123472690582275, -16.319135665893555, 71.98934173583984, -23.891748428344727, 18.04098892211914, 1.5810067653656006, -0.7495748400688171, -27.29739761352539, -27.878860473632812, -1.7150135040283203, -24.991565704345703, 16.076631546020508, 0.436800092458725, -7.069419860839844, -27.511693954467773, 11.845220565795898, -26.987348556518555, -45.70973587036133, 31.723621368408203, -1.2400940656661987, 5.002502918243408, 58.634979248046875, 9.65628433227539, -2.6825156211853027, -49.79061508178711, -26.642654418945312, 21.49188232421875, 58.25688171386719, -2.2835488319396973, 0.9396246671676636, 28.706714630126953, -10.767679214477539, -16.22530746459961, 37.04197311401367, -13.833874702453613, 18.615097045898438, 19.16658592224121, 31.37543296813965, 7.581295013427734, 60.87940216064453, -3.7946629524230957, -41.30097961425781, 12.673078536987305, -8.311504364013672, 20.6400146484375, 31.161781311035156, 19.496301651000977, -20.755136489868164, -45.99230194091797, 2.878241539001465, -39.784664154052734, -27.1368350982666, 51.50969314575195, -40.284019470214844, 17.73883819580078, 0.6794346570968628, 21.19066047668457, 23.990415573120117, -30.31363296508789, 5.556289196014404, -8.803719520568848, 32.71070861816406, 38.69978332519531, 54.0521240234375, 11.467194557189941, -7.009693622589111, 16.37674903869629, -35.986934661865234, -40.54029083251953, 12.93775463104248, -62.19225311279297, -18.41317367553711, 45.05414962768555, 23.883716583251953, 84.05927276611328, 53.863468170166016, 5.326985836029053, 18.065330505371094, 60.7840576171875, -5.246084213256836, 74.10601043701172, -35.704200744628906, 52.7780876159668, 30.719125747680664, 49.0837287902832, 34.47581100463867, 12.110532760620117, -0.7438335418701172, 8.320907592773438, 3.103424072265625, 13.524284362792969, 35.31514358520508, 6.127091407775879, -26.242002487182617, -27.22914695739746, -20.979780197143555, -40.276771545410156, -58.20830154418945, -48.987083435058594, 43.51164245605469, -15.758971214294434, -13.762358665466309, 19.368240356445312, -26.678274154663086, -3.0560965538024902, 25.2574462890625, -17.317440032958984, -40.5804443359375, 5.307209014892578, -3.064577341079712, 9.584177017211914, 16.31023597717285, -37.619842529296875, 64.82462310791016, 28.24403953552246, -9.275341033935547, 13.170425415039062, 45.686439514160156, -18.490896224975586, 8.661370277404785, -0.7704803943634033, -73.14238739013672, 13.176383972167969, 27.177614212036133, -23.363527297973633, -39.42097854614258, -39.04500961303711, -27.442626953125, -24.06822967529297, -18.560794830322266, 1.4912853240966797, -59.50222396850586, 84.89962768554688, -36.920501708984375, -10.032939910888672, 14.356158256530762, -32.71104049682617, -5.054018020629883, 36.24388122558594, -18.586652755737305, -40.39954376220703, -49.80436706542969, -45.26369094848633, 6.99605131149292, -14.396977424621582, 17.682615280151367, 55.420352935791016, -17.21940040588379, 1.3670873641967773, -34.3853759765625, 47.91337585449219, -2.8926949501037598, 6.414422988891602, 33.83186721801758, 75.40644073486328, 71.72731018066406, -26.7902774810791, -62.393218994140625, 1.8250666856765747, -31.37767219543457, 43.936771392822266, 44.17524719238281, 8.285079956054688, 0.5500338077545166, -33.007843017578125, 25.942163467407227, 11.010509490966797, 16.734939575195312, -20.045740127563477, -54.65054702758789, -9.712309837341309, -49.06988525390625, -2.6783366203308105, 13.039021492004395, 3.875340223312378, 25.429723739624023, -3.5716099739074707, -43.882328033447266, 15.353094100952148, 12.059555053710938, 15.73607349395752, 26.959548950195312, 20.020708084106445, 47.03400421142578, -29.969722747802734, -35.8355827331543, -44.085079193115234, 16.40681266784668, -25.696489334106445, -40.115333557128906, 39.77922439575195, 28.34725570678711, -30.275821685791016, -7.902212619781494, 44.31077194213867, 36.61264419555664, 38.30952835083008, -9.82491397857666, -48.87744903564453, -0.9203420877456665, -56.51063919067383, -13.10476016998291, -36.35193634033203, -10.241796493530273, -13.603065490722656, 0.6274915933609009, 15.472288131713867, 3.139641523361206, -9.078452110290527, 65.9033203125, -11.249211311340332, -27.409753799438477, 20.164623260498047, 3.021979808807373, 48.06681442260742, -54.217796325683594, -9.575034141540527, 3.374366283416748, -13.965020179748535, 33.79737854003906, -1.7102833986282349, -0.17393076419830322, -17.22510528564453, 56.288997650146484, 12.497737884521484, -31.070146560668945, -19.3824462890625, -28.175439834594727, -0.238736093044281, -16.15204429626465, 9.6745023727417, -9.255029678344727, 28.356544494628906, -14.830362319946289, -3.347259283065796, -5.637155532836914, -1.1176632642745972, -18.703739166259766, -8.265617370605469, 31.74994659423828, -60.476959228515625, 7.03894567489624, 13.810037612915039, -23.455677032470703, -58.604583740234375, 38.504127502441406, -40.3055419921875, 2.983116626739502, 13.907241821289062, -26.42879295349121, 27.675411224365234, 24.566753387451172, -24.3618106842041, 12.411659240722656, -2.142946481704712, 0.2028188705444336, 23.737960815429688, 21.339706420898438, 11.639012336730957, 23.05101203918457, -29.437591552734375, -35.3909912109375, 30.269092559814453, 0.4353569746017456, -24.45534324645996, -13.215513229370117, -19.093475341796875, 12.425897598266602, -34.01408386230469, 21.5179386138916, -7.748780727386475, -4.884939193725586, 11.913607597351074, 29.437387466430664, 41.17678451538086, 32.90080261230469, -19.402122497558594, -15.82876968383789, 38.98453903198242, -27.0544490814209, -7.160784721374512, -26.816381454467773, -11.11362361907959, -1.0945091247558594, -27.070470809936523, -41.01021194458008, -11.96323013305664, -54.040035247802734, -58.538116455078125, -12.090056419372559, 35.593502044677734, 29.77654457092285, -23.034042358398438, -4.753574371337891, 0.6784019470214844, -3.6607489585876465, -0.4596943259239197, -22.739776611328125, 12.593485832214355, 27.559885025024414, 21.065034866333008, 49.23671340942383, -32.15773010253906, -14.19549560546875, 8.934454917907715, 19.38730239868164, 8.698328018188477, 36.2098503112793, 24.640911102294922, -17.33909034729004, -61.593238830566406, 23.979976654052734, 4.364855766296387, -17.237287521362305, -28.3951416015625, -11.478941917419434, -19.654512405395508, -13.087733268737793, -47.71114730834961, 34.45283508300781, 28.41026496887207, 38.495025634765625, -57.87549591064453, -18.907482147216797, 23.117019653320312, 16.956396102905273, -8.884563446044922, 26.031118392944336, -12.434660911560059, 12.970437049865723, -18.3671817779541, 0.045018136501312256, -0.9028761386871338, -79.00923919677734, 64.2615737915039, -6.076113224029541, -59.976783752441406, 37.57969665527344, 0.4397057294845581, 40.824119567871094, -47.453147888183594, -35.50730514526367, 9.039692878723145, -28.813194274902344, -16.1158504486084, 16.970109939575195, 4.79862117767334, -5.877370357513428, 16.03042984008789, -6.400821208953857, -18.45905876159668, 43.34779357910156, -41.52840805053711, 75.51009368896484, 31.550676345825195, -11.936120986938477, 5.738613128662109, -10.474324226379395, -15.89285945892334, 25.99161720275879, -77.80310821533203, 10.759815216064453, -34.97815704345703, -19.565387725830078, 23.13882064819336, -10.45857048034668, -14.081204414367676, -89.3470687866211, -43.77085494995117, 15.226906776428223, -61.78588104248047, -58.97974395751953, -12.339847564697266, 23.7674560546875, 34.27458953857422, -37.56111145019531, 27.128124237060547, -18.629514694213867, 12.582219123840332, -28.683317184448242, -48.00416946411133, 13.137621879577637, -5.635410308837891, -13.068416595458984, -26.55161476135254, 57.94343185424805, 17.981853485107422, -12.784130096435547, 53.94597625732422, -23.941923141479492, 14.760936737060547, -11.820721626281738, -6.290470600128174, -20.03155517578125, 4.479737758636475, -23.056671142578125, -1.847900629043579, 0.3820003271102905, 82.88725280761719, -54.33125686645508, -29.990325927734375, 31.116233825683594, -6.618358135223389, -19.48902702331543, 9.541891098022461, 6.252442359924316, -28.20228385925293, -7.845609664916992, 33.203800201416016, -17.694923400878906, 16.621788024902344, -3.6648237705230713, 7.258495330810547, 15.456517219543457, -9.324895858764648, -6.862381458282471, -14.399020195007324, 12.789268493652344, -11.384286880493164, 11.31286907196045, 9.445599555969238, 8.970154762268066, 0.1820087432861328, 34.34089279174805, 12.720759391784668, 4.4394049644470215, -40.2666130065918, -42.18177032470703, 2.863347053527832, -34.9764404296875, 24.43645477294922, 16.278711318969727, -2.1415274143218994, -2.2537002563476562, 4.200338840484619, -22.474611282348633, 3.8487610816955566, 8.519328117370605, 43.722904205322266, 9.109528541564941, 11.939386367797852, 40.35505294799805, -47.47221374511719, -32.14369201660156, 19.85677146911621, 61.38970184326172, 22.412607192993164, 2.1568100452423096, 19.643367767333984, -17.562110900878906, 24.012609481811523, -22.739892959594727, -14.766416549682617, -34.89894104003906, 1.6690729856491089, 9.54305648803711, 34.75102615356445, -16.06533432006836, 22.418743133544922, -31.682830810546875, -24.35784339904785, -17.521316528320312, 17.44133186340332, 3.901017665863037, 9.476603507995605, -13.387357711791992, -21.20785140991211, -0.6525483727455139, 17.436492919921875, -26.63827896118164, -29.297794342041016, 3.5030341148376465, 45.765464782714844, -49.82573318481445, -12.43352222442627, 36.98344802856445, -16.518529891967773, 8.308971405029297, 24.011947631835938, -49.611412048339844, -0.9061833620071411, 17.095279693603516, 49.888343811035156, 38.46381759643555, 14.196076393127441, -2.4559366703033447, 11.899114608764648, 28.253963470458984, 49.83838653564453, 21.544599533081055, 26.889577865600586, -25.867002487182617, -18.482070922851562, -39.82978820800781, 65.71582794189453, -1.1812171936035156, -44.7434196472168, 0.7754209041595459, -12.466713905334473, 32.957008361816406, -3.766936779022217, -13.132560729980469, 0.42590421438217163, -48.884666442871094, 7.485755920410156, -54.1406135559082, 5.697338581085205, 66.7509536743164, -4.150456428527832, 39.926612854003906, 1.4145053625106812, 45.232879638671875, -5.037778377532959, -8.3702974319458, 31.566436767578125, -27.46580696105957, 6.19415807723999, -18.762062072753906, 43.35563278198242, 65.8789291381836, 30.365306854248047, 42.3378791809082, 35.324161529541016, -2.3222241401672363, -18.40619659423828, -24.751846313476562, -46.91633224487305, 15.474146842956543, 14.143780708312988, -23.948471069335938, 0.1725279688835144, 1.8832154273986816, 4.846698760986328, 4.66294527053833, -12.313276290893555, -30.57452964782715, -66.30559539794922, -31.34065818786621, -11.590821266174316, 65.67597198486328, -50.875762939453125, -15.225215911865234, 39.5026969909668, -44.92216110229492, 9.410494804382324, -44.233238220214844, -3.3398168087005615, 11.631002426147461, -19.78892707824707, -8.83570671081543, 8.788609504699707, -10.066134452819824, -3.7734456062316895, 7.305007457733154, -13.361446380615234, -6.207064628601074, 18.428672790527344, -27.79096794128418, 10.291264533996582, -33.5972785949707, 5.872854709625244, 13.732969284057617, 17.070133209228516, 11.540485382080078, -15.236473083496094, -20.06293487548828, -71.06069946289062, 40.81748962402344, 9.198114395141602, 3.815730333328247, -13.91882610321045, -44.689537048339844, -5.3867034912109375, 22.020471572875977, 47.0432014465332, 15.502084732055664, 31.1745662689209, 9.863778114318848, 25.183048248291016, 40.36097717285156, 28.973838806152344, -40.544036865234375, 3.3212573528289795, -8.01978588104248, -29.233543395996094, -14.536003112792969, 8.444028854370117, -10.25723648071289, -13.711082458496094, -4.838460922241211, 33.344459533691406, -14.690079689025879, -33.25784683227539, 21.881715774536133, 3.9451382160186768, 21.868309020996094, 0.7994133234024048, -62.00611114501953, -27.15134048461914, 35.58037567138672, 21.793128967285156, 40.47970199584961, 71.86286926269531, -2.8840231895446777, 45.61001968383789, 0.6931407451629639, -13.827729225158691, -7.256304740905762, 13.834754943847656, 54.48549270629883, -61.04808044433594, 36.98084259033203, 8.89863109588623, -38.3935661315918, -33.048919677734375, 14.614218711853027, -67.32304382324219, 57.03816223144531, 32.317161560058594, 46.04955291748047, -36.034996032714844, -20.887414932250977, -0.3443770110607147, 65.28240966796875, -19.657644271850586, -20.66272735595703, 44.92081832885742, -34.992835998535156, -22.20282745361328, -7.762861728668213, 29.915786743164062, -30.300846099853516, 23.502212524414062, -13.750288963317871, 22.214418411254883, 0.40436026453971863, -12.374823570251465, -10.059901237487793, 30.49810218811035, -12.247583389282227, -13.2232084274292, -18.739660263061523, 9.626985549926758, -54.1273078918457, 18.79309844970703, -12.778066635131836, 4.701351642608643, -32.095314025878906, 22.700672149658203, -32.244873046875, 0.13485872745513916, 19.49489402770996, 33.39512634277344, 8.026132583618164, 30.361656188964844, 23.710630416870117, 28.701711654663086, 4.440756320953369, 21.51355743408203, -46.21468734741211, -18.959388732910156, -5.272258758544922, 14.789819717407227, -29.435882568359375, -13.137988090515137, 37.04951858520508, -23.06627655029297, 7.149677276611328, -35.15163803100586, 47.48906707763672, 12.429181098937988, -49.283973693847656, -0.7754402160644531, 38.1026611328125, -7.164783000946045, 4.645071029663086, 14.188502311706543, -19.253084182739258, 12.120256423950195, 9.971617698669434, -24.0277099609375, -26.05542755126953, 15.467575073242188, -2.5624194145202637, 44.5584831237793, -24.18613052368164, 18.153112411499023, -38.512577056884766, -15.258495330810547, 1.201537847518921, 6.058964729309082, -11.131936073303223, 33.57769012451172, -31.94729995727539, 20.714420318603516, 20.70509147644043, 57.26055145263672, -1.3305349349975586, 49.44623565673828, -39.376277923583984, -36.171058654785156, -0.735785186290741, -33.23795700073242, -7.2515153884887695, -33.48135757446289, -43.13962936401367, -32.674312591552734, -50.367523193359375, -1.7820159196853638, -1.8200912475585938, 51.88089370727539, 53.456356048583984, 7.079019546508789, -41.03858947753906, -51.609378814697266, 9.721542358398438, 0.38491368293762207, 36.12094497680664, -12.397778511047363, -15.10893726348877, 37.53523254394531, -11.260851860046387, 12.47292709350586, -34.49128341674805, 56.02444839477539, 9.670300483703613, -43.95415496826172, 6.308923721313477, 34.89563751220703, 19.487180709838867, 13.279526710510254, 30.309215545654297, -14.746376991271973, -16.452394485473633, -56.093528747558594, 3.372509717941284, -26.509960174560547, 31.49126625061035, -4.185433387756348, 33.688045501708984, 2.385648727416992, -0.1940324306488037, 14.566954612731934, 28.433326721191406, 30.43173599243164, 17.689998626708984, -9.006880760192871, 1.029271125793457, 4.218705177307129, -16.486597061157227, -13.629827499389648, -23.136919021606445, -34.46177673339844, -4.013564586639404, 40.81197738647461, -5.024722099304199, -4.4804463386535645, -24.615201950073242, -20.609485626220703, -18.410289764404297, 21.255083084106445, 4.982125282287598, 43.13679122924805, 5.14433479309082, 24.5271053314209, -41.097511291503906, 32.62015914916992, -37.115936279296875, 6.3153839111328125, -37.734100341796875, -27.04484748840332, -27.535367965698242, -28.563861846923828, 72.84752655029297, -3.7892794609069824, -2.5189924240112305, -7.608611106872559, -16.220733642578125, -62.45957946777344, 18.41466522216797, 58.98986053466797, 11.44913387298584, -27.848060607910156, -25.71085548400879, -26.770475387573242, 18.694122314453125, -34.03248977661133, -12.365920066833496, 2.566903591156006, -3.4005162715911865, 56.549232482910156, -56.34383010864258, -15.810291290283203, -33.832332611083984, -17.192625045776367, -7.598426818847656, -14.318476676940918, 7.035656452178955, -5.987942695617676, 32.540283203125, -19.584501266479492, -1.897228479385376, 29.14225959777832, -30.52581787109375, -6.340158462524414, -17.340356826782227, 30.049463272094727, -8.733918190002441, -16.169824600219727, 48.75106430053711, -23.838083267211914, 2.127674102783203, -61.74894714355469, 9.886176109313965, -0.5229321122169495, 95.23989868164062, -27.527809143066406, 35.98459243774414, 4.7015790939331055, 16.24855613708496, -14.523895263671875, 28.304777145385742, 33.66438674926758, -29.21723747253418, -59.594818115234375, 28.981565475463867, 14.087790489196777, -4.3452372550964355, 7.555856704711914, -46.24520492553711, -31.28603172302246, 38.90761184692383, -11.743318557739258, -36.248355865478516, -26.367624282836914, -6.976871967315674, -1.109722375869751, 13.441214561462402, 63.34772491455078, 28.391630172729492, 0.5723169445991516, 0.7303237318992615, -3.644432783126831, -34.165218353271484, -7.363399028778076, 59.11069869995117, -4.385483741760254, -65.29802703857422, -14.080385208129883, 11.750870704650879, 38.5648078918457, 12.094770431518555, 11.20995807647705, 38.89424133300781, 3.831589937210083, 21.419540405273438, 7.433694362640381, 0.6426427364349365, -38.570220947265625, -3.6277015209198, 6.858587265014648, -8.348146438598633, 20.681350708007812, -25.999258041381836, -14.474248886108398, -50.41332244873047, 33.33829879760742, 7.193926811218262, 50.82123947143555, 11.40198040008545, -21.03345489501953, 14.955979347229004, 24.2817440032959, -25.037294387817383, 28.557323455810547, -31.229690551757812, 32.362545013427734, -25.140289306640625, 3.251032590866089, 47.201663970947266, 2.110003709793091, 20.476634979248047, 6.855904579162598, 26.230405807495117, 33.697998046875, -24.16797637939453, -20.14661407470703, 18.58909797668457, -44.92271423339844, -34.417015075683594, -47.0430793762207, -45.50366973876953, 0.6038304567337036, 28.687469482421875, -30.78129005432129, 38.93043899536133, 10.648996353149414, 34.97658920288086, -4.520328044891357, 23.483978271484375, 53.307735443115234, -18.226076126098633, -18.4188289642334, -23.938549041748047, 6.289697170257568, 14.970098495483398, 3.220877170562744, 52.21110534667969, -16.754133224487305, 24.52440071105957, 21.06801414489746, -14.683133125305176, -37.372779846191406, 30.19243049621582, 48.00648880004883, 47.53547668457031, 11.383894920349121, -26.9416446685791, -5.777997970581055, 9.556492805480957, -64.54960632324219, -7.669044494628906, 68.45735168457031, 18.883132934570312, 35.25965118408203, -24.57232093811035, -2.9770984649658203, 14.122377395629883, 24.93780517578125, 62.819297790527344, 74.34416198730469, -18.089460372924805, 38.95765686035156, 14.238021850585938, -25.881305694580078, 38.935115814208984, 9.66794490814209, 40.2870979309082, 0.270526647567749, -17.274978637695312, -1.0389909744262695, 65.8210678100586, -3.979379415512085, -24.900115966796875, -5.808676719665527, 32.75233459472656, 7.132482528686523, -46.181610107421875, -3.430189847946167, 21.382844924926758, 2.6971497535705566, 25.10477066040039, -13.702431678771973, 20.449920654296875, 17.1538028717041, 13.105271339416504, -6.161701679229736, 65.17425537109375, -10.771245002746582, 5.544338703155518, -15.295964241027832, -4.091509819030762, -70.14396667480469, 10.672935485839844, 0.23130720853805542, 14.18179702758789, -14.024467468261719, 23.107181549072266, -22.508760452270508, -17.36031723022461, 22.549728393554688, -48.7765007019043, -28.504064559936523, -43.9063720703125, 17.046733856201172, -15.674481391906738, 20.204679489135742, 7.346158981323242, 32.97867202758789, 1.034837007522583, 5.820650577545166, -22.313480377197266, -36.625526428222656, -0.4643952548503876, -36.00002670288086, 5.473050594329834, -29.675209045410156, 9.650701522827148, 29.391674041748047, -36.21880340576172, -12.580768585205078, 10.402849197387695, -4.931143283843994, -20.524066925048828, 20.116483688354492, -4.6265387535095215, 37.099952697753906, -34.589454650878906, -52.345516204833984, -8.277332305908203, 2.0840022563934326, 19.08171844482422, 31.6374454498291, 30.572368621826172, 64.31106567382812, 14.48055362701416, -0.29720649123191833, 13.035648345947266, -6.197350978851318, 40.06544494628906, -6.396154403686523, -26.928993225097656, -6.284579753875732, -20.09769058227539, -6.546833515167236, -39.393558502197266, 30.10885238647461, 17.914321899414062, -16.069684982299805, 1.922469973564148, 43.27706527709961, 11.358963966369629, 9.10274887084961, -10.164668083190918, -38.734336853027344, -13.14037799835205, 61.699668884277344, -10.932540893554688, -9.709696769714355, -11.920769691467285, 18.284997940063477, -76.35395812988281, -22.79330825805664, -27.819385528564453, -40.250022888183594, 19.071128845214844, -15.847618103027344, 10.975556373596191, 26.91504669189453, -15.597413063049316, 7.205542087554932, 31.535831451416016, -4.492694854736328, 22.110435485839844, -31.902816772460938, -9.945537567138672, -29.7253360748291, -0.8108211159706116, -51.50972366333008, -4.494445323944092, -33.95924377441406, -30.322742462158203, 47.644187927246094, -14.200640678405762, -4.57103967666626, -26.306428909301758, -63.71009063720703, 16.27912712097168, -17.05475616455078, 1.4142471551895142, 43.51821517944336, -13.639561653137207, -4.15623664855957, 7.761557102203369, -22.49124526977539, -28.621370315551758, -15.818098068237305, -1.3279366493225098, -39.01347732543945, -38.97174835205078, 10.962305068969727, 11.315577507019043, -13.512951850891113, -17.182403564453125, -3.613966941833496, 10.732695579528809, 35.0771598815918, -7.260080337524414, 17.179698944091797, 4.640444755554199, -0.3555295467376709, -46.09239196777344, -5.799737930297852, 40.03028869628906, -30.04236602783203, 4.285369873046875, 45.456478118896484, 0.03679278492927551, -87.9303970336914, 45.369834899902344, -70.40726470947266, -20.30949592590332, -14.193327903747559, -24.417329788208008, -4.738115310668945, 10.793549537658691, 5.300243854522705, -63.28147888183594, 6.306236267089844, -28.713621139526367, 17.284690856933594, 42.10762405395508, -37.15003967285156, -26.680646896362305, -14.788726806640625, -48.14650344848633, -6.821900844573975, -41.53907775878906, 58.38279724121094, 7.044294357299805, 72.24443054199219, -37.02498245239258, -46.436824798583984, 27.3240909576416, -4.90487003326416, 45.66679763793945, 43.135032653808594, 21.785856246948242, -32.24803161621094, -2.9163033962249756, 18.445249557495117, 8.722342491149902, 18.55705451965332, 40.682212829589844, 10.88958740234375, -64.47384643554688, -17.27239990234375, 14.489598274230957, 19.600881576538086, 77.79541778564453, 49.39736557006836, -61.86925506591797, -3.8604063987731934, 21.91166114807129, 15.861576080322266, -21.663297653198242, -21.933860778808594, 2.3165996074676514, -12.150931358337402, -49.09849166870117, 9.530187606811523, 7.607673645019531, -2.904534339904785, -13.96774673461914, 46.05122756958008, -20.358341217041016, 18.620044708251953, 40.484474182128906, -26.974674224853516, -24.22555923461914, 4.355292797088623, 2.9556684494018555, 22.620874404907227, -27.55930519104004, -22.31523323059082, -20.088708877563477, 28.183074951171875, -2.3172571659088135, -1.117017149925232, -70.1846694946289, -2.97824764251709, 12.009102821350098, -7.096718788146973, -19.393802642822266, 37.6535758972168, 12.652198791503906, 20.73748016357422, -48.17972946166992, 46.768733978271484, -39.137168884277344, 17.59014892578125, -15.302247047424316, -47.08580017089844, -24.63900375366211, 49.991416931152344, 34.47723388671875, -39.95747756958008, -5.2344136238098145, 6.8508453369140625, -5.096098899841309, 7.303554058074951, 44.9326057434082, -51.00142288208008, 6.40472412109375, -51.011131286621094, 29.797365188598633, -13.043489456176758, -0.8548297882080078, 14.753111839294434, -16.120033264160156, -19.17292594909668, 12.507951736450195, 5.564416408538818, -51.50224685668945, -14.226419448852539, 4.914235591888428, 15.797308921813965, -28.539640426635742, 19.52522087097168, 8.260332107543945, -23.959228515625, -56.31787109375, 6.332680702209473, -2.358412742614746, 7.509383201599121, 20.926136016845703, -4.490272521972656, 7.046997547149658, -28.894922256469727, 3.3189284801483154, -36.55006408691406, -11.879502296447754, 15.72180461883545, -8.268878936767578, 50.30421829223633, -48.7236442565918, -20.46246337890625, -12.77737808227539, 20.513551712036133, 32.53693771362305, 16.94950294494629, -5.30817985534668, 33.4754524230957, -25.389150619506836, -32.82911682128906, -29.659584045410156, -3.185894250869751, 2.928706645965576, 55.866573333740234, -29.664443969726562, 1.689652442932129, -45.72050857543945, -18.259937286376953, 69.32305145263672, -23.914899826049805, -25.51589012145996, -6.770058631896973, -36.05622482299805, -43.74211120605469, -39.523765563964844, -24.547704696655273, 26.744483947753906, 34.87295913696289, 45.67229080200195, 72.47897338867188, -30.349918365478516, -20.742347717285156, -17.626863479614258, -49.61988830566406, -39.01631164550781, 20.58783721923828, -12.553509712219238, 6.789243221282959, 36.31368637084961, -15.633077621459961, -39.44664764404297, 7.343379497528076, -10.968217849731445, -6.356509208679199, -33.22113800048828, 11.303638458251953, -13.687028884887695, 13.97427749633789, 13.416071891784668, -18.156335830688477, 4.7370500564575195, -21.734546661376953, 21.2620849609375, 18.657461166381836, -29.54067039489746, -31.77871322631836, -26.30501937866211, 58.117313385009766, 12.126588821411133, -29.815135955810547, 16.44400978088379, -43.076942443847656, 30.449445724487305, 59.74969482421875, 3.8312082290649414, 6.917186737060547, 51.481300354003906, -16.49680519104004, 25.474775314331055, -1.3161733150482178, 7.699892520904541, 6.634270668029785, 5.4418721199035645, 68.412109375, 20.682676315307617, -39.08650207519531, -88.95941162109375, 14.102325439453125, -36.364646911621094, 1.4381606578826904, 15.821788787841797, -36.91511917114258, 22.898229598999023, 25.688751220703125, -18.949188232421875, 26.6112060546875, -19.787830352783203, -1.2748572826385498, -24.78997802734375, 17.10713768005371, 47.646366119384766, 14.496395111083984, 12.103692054748535, -0.5597031116485596, -68.47594451904297, 41.489078521728516, 22.27640151977539, -30.4190673828125, -11.76871395111084, -28.494808197021484, -57.73057174682617, 63.24293518066406, 27.995084762573242, -22.7485408782959, 30.81297492980957, 34.87104415893555, -4.583929061889648, 4.836930751800537, -1.2796162366867065, -49.70940399169922, -9.454815864562988, 32.20314025878906, 22.420732498168945, -55.98173522949219, -5.035396575927734, -39.345497131347656, -30.86509132385254, -17.944988250732422, 10.892733573913574, -56.108158111572266, 35.124473571777344, -32.09221267700195, -10.574040412902832, 18.041515350341797, -16.520179748535156, -23.622610092163086, -12.743232727050781, -31.03364372253418, -15.06934928894043, -7.55150032043457, -30.201356887817383, -2.7120182514190674, 14.49179458618164, -23.32322883605957, 0.70377117395401, -47.14749526977539, 15.368583679199219, 12.645646095275879, -33.21758270263672, 2.94976544380188, -35.28910827636719, 2.562004566192627, -57.552940368652344, -9.286002159118652, 8.566925048828125, -0.24928009510040283, -31.027454376220703, -27.915613174438477, 11.117223739624023, -5.699871063232422, -14.069570541381836, 22.561819076538086, -37.913299560546875, -30.52442169189453, 27.922161102294922, -3.644845724105835, 7.920852184295654, 13.535776138305664, -1.2473437786102295, -3.3391950130462646, -0.7344721555709839, 22.22550392150879, -15.182605743408203, 14.37914752960205, -3.1902434825897217, 27.819936752319336, 7.967007637023926, 15.723714828491211, -19.07537841796875, 28.50515365600586, -40.301475524902344, -31.884416580200195, 13.760978698730469, -0.5135392546653748, 16.728031158447266, 9.893248558044434, 15.40060806274414, -55.67795181274414, 2.3610472679138184, 0.1620437502861023, 26.11896324157715, -28.978435516357422, -11.29358959197998, 15.936567306518555, 29.28228759765625, -68.02412414550781, -4.193813800811768, 14.58066463470459, 48.24488067626953, -26.072046279907227, 11.172323226928711, -49.10033416748047, 78.68123626708984, -14.230744361877441, -5.366534233093262, -21.89556884765625, -13.396005630493164, -0.5389420986175537, 19.953277587890625, 42.860050201416016, 20.985727310180664, -13.393271446228027, 33.88410568237305, -11.05862045288086, 14.636600494384766, 43.93937301635742, 40.644012451171875, 56.09524917602539, -30.16053009033203, -15.959144592285156, -12.473014831542969, -42.434967041015625, -41.65171813964844, -18.047710418701172, 10.395584106445312, 50.49544143676758, -1.296259880065918, -14.47583293914795, -11.972661972045898, 10.42806339263916, 52.83934783935547, -17.231163024902344, -5.230651378631592, -8.573843955993652, 14.063653945922852, 3.180032968521118, 7.729248046875, 13.06999397277832, 3.5282115936279297, 8.476981163024902, -27.02714729309082, 19.912179946899414, -5.095530986785889, 21.31578826904297, -48.525856018066406, -14.963507652282715, 49.798248291015625, 9.685375213623047, 11.638792037963867, -15.07812213897705, 55.80883026123047, -0.6829608082771301, 34.122406005859375, -23.086027145385742, 51.02353286743164, 8.136383056640625, 3.4488048553466797, 28.398759841918945, -17.97854995727539, -51.82075881958008, -0.6973633766174316, 26.473386764526367, 20.922975540161133, 5.662372589111328, -7.440842151641846, -13.700431823730469, 45.723663330078125, -23.920284271240234, 12.625628471374512, 22.31256103515625, -37.072662353515625, 29.800403594970703, 27.768457412719727, -11.117512702941895, -3.2294764518737793, 31.84775733947754, 24.172456741333008, -33.74046325683594, -3.5535190105438232, -49.050228118896484, 30.019197463989258, -5.458251476287842, -29.280088424682617, 23.223052978515625, -19.37476348876953, 40.18009567260742, 43.627140045166016, 37.85358428955078, 30.690948486328125, -82.06000518798828, -24.80755615234375, -64.31013488769531, 19.73114776611328, -49.31802749633789, 6.322078704833984, -49.0042610168457, 10.860626220703125, -26.8599853515625, -51.39652633666992, -57.95476531982422, -13.99179744720459, 42.67041015625, -4.480062007904053, 47.309661865234375, -11.597077369689941, -0.07706320285797119, 20.82596778869629, 7.164984226226807, -24.403173446655273, -16.128246307373047, 13.874777793884277, 16.268604278564453, 14.170042991638184, 32.272422790527344, -1.0809884071350098, 1.425343632698059, 13.82669448852539, -8.407609939575195, -33.313846588134766, -9.665963172912598, -6.367204666137695, 32.85093307495117, -20.35936737060547, 20.01130485534668, 51.533546447753906, 11.82275676727295, -24.209442138671875, -17.491085052490234, -18.715896606445312, -20.65053939819336, -46.96880340576172, 4.559806823730469, 20.793107986450195, -8.9292573928833, -4.009115695953369, 6.550942897796631, -15.80526065826416, -19.53028678894043, 34.516727447509766, 17.342897415161133, -30.58119010925293, -20.442245483398438, 29.154712677001953, -79.60469818115234, -65.68405151367188, 11.0631103515625, 61.464630126953125, 17.075891494750977, 57.58511734008789, 82.26182556152344, 21.149198532104492, -44.720977783203125, 11.954354286193848, -26.24112319946289, 1.1604293584823608, 2.809396743774414, -3.7222037315368652, -59.50189971923828, -34.95734786987305, 34.696964263916016, 9.445549011230469, -4.698335647583008, 15.197481155395508, 4.947957515716553, 34.514644622802734, 24.596336364746094, -17.97344207763672, -64.99368286132812, 61.44758987426758, 2.713340997695923, 19.94579315185547, 2.6137630939483643, -36.91957092285156, -12.290861129760742, 12.480385780334473, 4.795003414154053, -11.655959129333496, 61.903778076171875, 27.365877151489258, 17.524702072143555, 6.961851119995117, 17.526195526123047, 11.898021697998047, -30.516151428222656, 21.997669219970703, 0.41834044456481934, -19.481353759765625, 7.845702171325684, 55.10966491699219, -7.609853744506836, -57.03205490112305, -5.173665523529053, 40.106048583984375, -14.10228157043457, 26.291839599609375, -15.067233085632324, -4.609710693359375, 40.92192840576172, 32.056461334228516, 14.494757652282715, 2.537358283996582, -48.933006286621094, -10.82156753540039, -15.959747314453125, -3.4267923831939697, -0.932727575302124, -2.373091459274292, -20.14667510986328, 6.357269763946533, 15.415596961975098, 107.56721496582031, 17.906103134155273, 15.554000854492188, -29.19156837463379, 60.500709533691406, 16.240026473999023, 56.09845733642578, 51.62942123413086, -34.24351501464844, -14.636678695678711, 23.270904541015625, -49.576377868652344, 2.385244607925415, 37.366371154785156, -49.049095153808594, 15.54121208190918, -9.187849998474121, -29.974014282226562, -4.7770185470581055, -28.21808433532715, 54.98273468017578, 34.950496673583984, 10.915669441223145, -21.092418670654297, -37.94856643676758, 27.698984146118164, 30.199161529541016, 13.81474494934082, 19.521669387817383, -16.3216609954834, 13.728825569152832, -25.71356201171875, -4.274635314941406, 35.96661376953125, -9.083240509033203, -16.75494956970215, 29.906509399414062, -30.19281768798828, 33.5531005859375, -35.43436050415039, 10.753292083740234, -2.4937796592712402, -9.329686164855957, 23.146268844604492, -22.918128967285156, -48.45051193237305, -3.1116862297058105, -18.609268188476562, 28.54720687866211, -5.502254009246826, -17.063005447387695, -31.141422271728516, 2.2889223098754883, 8.33840274810791, 36.41279602050781, -18.544391632080078, -5.933191776275635, 3.298135757446289, 31.6971492767334, 65.69097900390625, -8.316333770751953, 26.694223403930664, 28.674816131591797, 28.832517623901367, -4.111008167266846, -17.285503387451172, 15.134390830993652, 3.5894219875335693, -6.960367202758789, -12.360527992248535, -3.0309958457946777, 14.210796356201172, -26.344024658203125, -26.77947998046875, 4.175509452819824, -12.633918762207031, -24.73265266418457, 5.967129707336426, 20.765575408935547, 5.053919792175293, 23.18468475341797, -21.039955139160156, -10.849122047424316, -13.674951553344727, 23.79990005493164, 15.495026588439941, 0.7190878987312317, 4.23015022277832, 2.159064769744873, -43.897674560546875, -34.29682159423828, -34.8641357421875, 5.316629886627197, -11.70567512512207, 4.439487934112549, -16.290437698364258, -43.537437438964844, -2.4599311351776123, -1.5091630220413208, -22.19473648071289, -7.811678409576416, -2.2708182334899902, -28.570585250854492, -57.71310806274414, -5.751735687255859, 15.3480224609375, -42.21238708496094, -37.687442779541016, -43.23806381225586, -23.986289978027344, -19.36151885986328, -47.82368469238281, -23.60586166381836, -17.416723251342773, -36.80732345581055, 38.97341537475586, 3.2487874031066895, 3.8409347534179688, -10.490059852600098, 18.130756378173828, -23.87041664123535, 21.04388427734375, 8.306742668151855, 9.927962303161621, -40.08017349243164, -9.724075317382812, 12.551651000976562, 81.54998016357422, 39.705718994140625, -23.498340606689453, 38.234806060791016, -29.193769454956055, -27.327083587646484, -51.55686569213867, 21.239910125732422, -5.0993852615356445, -23.52004051208496, 40.79869842529297, -5.6162004470825195, -57.598777770996094, 27.989830017089844, -48.77790451049805, 24.430543899536133, 28.526935577392578, -4.281069278717041, -60.12758255004883, -6.2147932052612305, -39.7741584777832, -10.08717155456543, 0.8752574920654297, -41.2557258605957, 7.244669437408447, 22.931737899780273, 11.044593811035156, -11.733288764953613, -62.63747024536133, -5.583316326141357, 31.62009048461914, 51.69881820678711, 9.138717651367188, -37.90017318725586, 12.91009521484375, 9.471513748168945, -2.2985048294067383, 35.83500289916992, -36.50993728637695, 7.626436233520508, -19.968414306640625, -25.190446853637695, -4.441335678100586, 36.47724914550781, -28.473405838012695, 62.24067306518555, -43.71638488769531, -12.612142562866211, 1.946084976196289, 2.1145405769348145, -62.206092834472656, -2.0899128913879395, 32.55025863647461, -5.268791675567627, -10.558219909667969, -50.609249114990234, 43.39104080200195, 29.539798736572266, 33.39592742919922, 28.908082962036133, -46.526615142822266, 19.634737014770508, -43.686344146728516, 4.592134475708008, -0.04597938060760498, -13.455707550048828, 9.249723434448242, 12.560624122619629, -16.9261417388916, 21.301218032836914, 24.03282928466797, 33.22833251953125, -23.755369186401367, 33.695011138916016, -35.23275375366211, -32.58409881591797, -39.01731872558594, -46.56682205200195, 25.070823669433594, -15.882569313049316, 11.723355293273926, -40.50516891479492, -22.081418991088867, 18.604106903076172, -37.03718566894531, -21.705760955810547, 31.9427547454834, -25.00310516357422, -55.30387496948242, -73.56310272216797, -24.215469360351562, -5.349062919616699, -33.743370056152344, -3.030216693878174, -34.82740020751953, 8.565357208251953, 15.305846214294434, -53.939735412597656, 30.868593215942383, -74.88328552246094, 52.47556686401367, -15.33890151977539, -57.635711669921875, 18.14729118347168, -29.55747413635254, 5.0414299964904785, 97.7193603515625, 2.580052614212036, -37.480247497558594, -22.884750366210938, -4.554107189178467, 29.69646453857422, 45.324058532714844, -20.792274475097656, -19.12778091430664, 6.402883529663086, 52.47892379760742, 40.08201217651367, 46.02472686767578, 12.772006034851074, 12.183837890625, 42.50072479248047, -25.874282836914062, -53.710567474365234, 5.121582508087158, -6.896286487579346, -1.924511432647705, -17.596467971801758, -10.214632034301758, -5.703795909881592, -43.646751403808594, -43.20724868774414, -19.900737762451172, -58.58174514770508, -28.23809814453125, -13.3060941696167, -14.345792770385742, -22.663110733032227, 21.31470489501953, 32.774986267089844, 11.9864501953125, -13.343132019042969, -7.613476753234863, -60.475101470947266, -12.698040008544922, 48.718772888183594, 16.572622299194336, -53.60359573364258, 36.8218994140625, 12.474875450134277, -22.206972122192383, -47.80847930908203, -16.511293411254883, -15.924345016479492, 10.318856239318848, 13.782670021057129, 13.404898643493652, 50.576297760009766, 5.16296911239624, -0.4496413469314575, -18.219209671020508, 32.78893280029297, 33.75988006591797, -24.953201293945312, 14.844524383544922, 27.825170516967773, 3.2408652305603027, -60.22500991821289, -2.654634475708008, -46.057151794433594, 27.496973037719727, 7.011538982391357, 39.005794525146484, 14.506338119506836, -48.07106018066406, -4.320919513702393, 47.800148010253906, 39.63364028930664, -13.461807250976562, -7.829493045806885, 25.50858497619629, -22.00637435913086, -26.731889724731445, -58.96745300292969, 2.2140727043151855, -2.2159759998321533, 42.863826751708984, 7.769331932067871, -8.979267120361328, -17.066688537597656, -18.533313751220703, 12.560831069946289, -53.361961364746094, -52.3843994140625, 40.26383972167969, -98.60855102539062, 11.6358060836792, 38.750274658203125, -57.096065521240234, -48.33553695678711, 56.46052169799805, 3.281506061553955, 7.908967018127441, 10.863612174987793, 13.294798851013184, -4.511848449707031, 55.2744026184082, -33.29548263549805, 7.123993873596191, 51.22754669189453, -37.125144958496094, -30.59770965576172, -21.954219818115234, 6.2982096672058105, 19.149229049682617, -29.252410888671875, 7.8939313888549805, 52.55886459350586, 26.108501434326172, -17.314529418945312, 7.214430332183838, 15.598762512207031, -37.950225830078125, -4.631720542907715, 29.98529052734375, 21.22575569152832, 22.68987464904785, -15.546082496643066, 0.4290556013584137, -59.616336822509766, 36.191715240478516, -19.839754104614258, 2.0259809494018555, 9.496750831604004, -71.89462280273438, 27.61350440979004, -12.21502685546875, 21.445388793945312, -46.891510009765625, 27.451995849609375, 24.112091064453125, 5.972799301147461, 12.101391792297363, -6.545444011688232, -2.6029675006866455, -9.66341495513916, 32.286346435546875, -29.644092559814453, -44.211585998535156, -24.051820755004883, 8.489137649536133, 1.5791062116622925, -1.206284999847412, 82.1760025024414, 20.16913414001465, -43.95685958862305, 30.9764404296875, 54.06315994262695, -11.512346267700195, 53.190162658691406, 52.49231719970703, 39.301795959472656, -26.391897201538086, 15.977375984191895, -31.948810577392578, -42.4745979309082, -32.96065139770508, -14.460963249206543, 3.647170305252075, 40.72019958496094, 30.49587631225586, 4.5327067375183105, -22.807979583740234, -19.49747085571289, -15.979124069213867, 20.297534942626953, -37.583011627197266, -44.38981246948242, 47.24201202392578, 18.050273895263672, -7.172996997833252, -22.02105712890625, 45.121944427490234, -10.30850887298584, -7.759446620941162, 27.188188552856445, 12.558418273925781, 30.48714828491211, -53.98735427856445, 25.78118133544922, 25.162338256835938, -21.130157470703125, 49.04331588745117, -7.483264923095703, 10.445932388305664, -19.176883697509766, -28.910730361938477, -8.023473739624023, -40.371395111083984, -23.300853729248047, -59.006717681884766, -35.655982971191406, -4.64154577255249, 3.7190656661987305, -18.7803955078125, -9.77808666229248, 6.580009460449219, -25.411304473876953, -61.86210250854492, 19.921415328979492, -5.482117176055908, 5.010080337524414, -66.783447265625, 1.7976311445236206, 33.05904769897461, 27.783353805541992, -0.05570507049560547, -32.07167434692383, -51.599334716796875, -9.69351863861084, -15.161124229431152, 31.137592315673828, 17.089815139770508, 0.34718096256256104, 16.067424774169922, 14.40611457824707, 12.969284057617188, 1.4183855056762695, 15.264938354492188, 7.017231464385986, 17.50035285949707, 19.729782104492188, -32.56903839111328, -2.864898681640625, 34.31465148925781, -14.550256729125977, 9.34455680847168, 27.81170654296875, -13.070987701416016, -7.111217021942139, 14.360482215881348, -50.59751510620117, -18.546199798583984, -17.00874137878418, 32.44382858276367, 20.386343002319336, 32.46364212036133, -7.4178056716918945, 11.64451789855957, 9.972864151000977, -13.32421875, 19.707847595214844, 4.28311014175415, -19.7789306640625, 20.364463806152344, -13.50776195526123, 63.110939025878906, -12.565237045288086, 60.21862030029297, 12.216349601745605, -15.092477798461914, -33.538360595703125, -37.752864837646484, 12.115215301513672, 76.57030487060547, -17.73295783996582, -14.637808799743652, -53.52596664428711, -33.256492614746094, -13.141383171081543, -8.402402877807617, -15.585314750671387, -27.031864166259766, -9.742241859436035, 28.52910041809082, 0.42683762311935425, -67.22525787353516, -9.560531616210938, -15.986202239990234, 2.133970260620117, 33.161224365234375, 22.07766342163086, -60.42380142211914, -18.777137756347656, 5.376697540283203, 0.9527425765991211, 28.505290985107422, -5.68361234664917, -20.428447723388672, -17.999523162841797, -26.450098037719727, 15.4972505569458, 63.203712463378906, 3.615452766418457, 3.5574262142181396, -34.966949462890625, 12.660722732543945, 1.5704002380371094, 10.874727249145508, -9.286091804504395, 31.77894401550293, 45.01777267456055, 16.51546287536621, -16.774070739746094, -46.38836669921875, -50.81840133666992, 2.6962666511535645, 41.340049743652344, -9.962715148925781, -34.73281478881836, 22.445751190185547, -15.175553321838379, 23.37748146057129, 31.882455825805664, 25.528446197509766, 44.85506820678711, -8.848967552185059, -16.659696578979492, -14.865510940551758, -41.85446548461914, -25.89732551574707, -6.491738319396973, -5.019980430603027, -35.638370513916016, 29.687602996826172, -41.73986053466797, -10.982583045959473, -47.431053161621094, -3.3388423919677734, -37.933834075927734, -40.72393798828125, 43.826011657714844, 19.27069664001465, -12.18714427947998, -16.922414779663086, 26.1107120513916, -11.436421394348145, -45.396080017089844, 31.07600975036621, 35.06495666503906, -3.2228641510009766, 9.18786334991455, 33.53670883178711, 6.801964282989502, 11.920833587646484, 34.21438980102539, 10.654789924621582, 40.10515594482422, -20.223834991455078, -11.696368217468262, -46.61350631713867, 24.46109390258789, 29.0301456451416, -18.63261604309082, -3.186422824859619, -0.5839830636978149, 33.952884674072266, -33.0132942199707, -18.06224250793457, -44.894569396972656, -32.87849426269531, -16.283241271972656, 6.823413848876953, -37.586021423339844, -8.976475715637207, 23.019514083862305, -40.32389831542969, 6.737485885620117, -20.18283462524414, 23.577932357788086, -15.784852027893066, 62.6162223815918, -8.600410461425781, -20.331388473510742, 37.932395935058594, -9.301499366760254, 26.030853271484375, 5.026579856872559, -13.611834526062012, -12.243429183959961, 23.76852035522461, 55.33661651611328, -14.46644115447998, 17.194183349609375, 27.534439086914062, -14.148982048034668, 45.92250442504883, 2.7645761966705322, 9.517460823059082, 8.413274765014648, -6.217854976654053, 10.777795791625977, -23.10784339904785, 39.75257110595703, -8.022134780883789, -6.863689422607422, -28.841352462768555, 56.424476623535156, 33.632568359375, -13.625911712646484, -34.37290954589844, 3.5358567237854004, 12.672812461853027, 26.68416976928711, -22.725427627563477, -22.257753372192383, -0.40621912479400635, 36.99333572387695, 55.038822174072266, -11.080135345458984, 2.7728047370910645, -22.377779006958008, 1.8064688444137573, -3.938141107559204, 18.81182861328125, -0.46655672788619995, -27.535619735717773, -18.457937240600586, 29.694875717163086, 18.8107967376709, 53.55884552001953, 14.219168663024902, 5.024017333984375, 28.98079490661621, 39.683258056640625, 13.908711433410645, 7.92336368560791, 30.32537269592285, -48.5234375, 26.380130767822266, -14.965203285217285, 57.00190353393555, -31.960037231445312, -3.516106605529785, -21.438196182250977, -46.04111862182617, -8.211620330810547, -14.778385162353516, -52.043121337890625, 1.021505355834961, -70.40605163574219, 18.190143585205078, -3.771494150161743, 3.952613592147827, -31.144550323486328, 8.365633010864258, -34.49009704589844, -17.384471893310547, -11.497300148010254, 20.45766830444336, -38.725830078125, -53.20384979248047, 17.773040771484375, -15.642447471618652, -10.139522552490234, 26.353757858276367, -0.34428176283836365, -19.3707332611084, 8.257576942443848, -55.41938018798828, 13.093676567077637, -33.20530700683594, -14.543646812438965, 14.176636695861816, -11.186387062072754, -40.293697357177734, -14.051965713500977, 23.88804817199707, -2.8873159885406494, 14.470510482788086, -25.82563591003418, -25.41170310974121, 2.1712050437927246, 12.307571411132812, 8.830768585205078, -0.8965340852737427, -17.657737731933594, 6.069270133972168, 50.0133171081543, -4.10315465927124, 1.984602689743042, 9.860132217407227, 42.9166145324707, 26.153827667236328, 36.18166732788086, -16.396108627319336, 28.96784019470215, -18.15182113647461, 34.24726867675781, -11.652956008911133, -26.18437957763672, 6.874085903167725, 44.777305603027344, 23.165390014648438, 16.84483528137207, -11.7044677734375, -22.636821746826172, 24.122798919677734, -12.138778686523438, -24.391948699951172, 40.82192611694336, 41.68772888183594, -5.810694694519043, 10.991547584533691, -5.510164260864258, 16.142250061035156, 4.574718952178955, 60.179420471191406, 1.1201376914978027, 32.761375427246094, -54.78836441040039, 6.02690315246582, 49.83478546142578, -15.133922576904297, 12.845536231994629, -26.3651065826416, -37.87449264526367, 16.353368759155273, -23.916458129882812, 19.104015350341797, -11.089628219604492, 16.553831100463867, 34.31910705566406, -15.691941261291504, -54.34098815917969, 0.6758162975311279, 0.2560661733150482, -10.377689361572266, -36.42225646972656, 3.5697431564331055, 58.53757095336914, -34.060203552246094, 34.72541809082031, 23.355600357055664, 8.688111305236816, 35.5518684387207, 0.04697465896606445, 38.27273941040039, 34.24011993408203, -15.208783149719238, 30.099018096923828, -27.84972381591797, -95.4539566040039, 16.87040138244629, 37.5331916809082, -11.318275451660156, -22.259754180908203, 8.742732048034668, 31.965599060058594, 1.6038398742675781, -16.812395095825195, -10.451678276062012, -32.661197662353516, -6.860256671905518, 22.89072608947754, 17.25324821472168, 9.08061408996582, -3.9413022994995117, 24.37317657470703, -24.701452255249023, 23.56699562072754, 17.006731033325195, -15.75539493560791, -28.0263729095459, 2.395136833190918, -50.450016021728516, -10.677131652832031, -69.38634490966797, -30.279563903808594, -40.3245735168457, 0.2309529185295105, 50.03452682495117, 4.927860736846924, 10.127068519592285, 27.394081115722656, 20.485361099243164, 26.897729873657227, -45.613685607910156, 62.52347183227539, -33.244930267333984, 51.714542388916016, -19.618846893310547, 8.017660140991211, 19.5256404876709, -10.145898818969727, -9.94095516204834, 27.06427574157715, 71.20496368408203, -5.371816635131836, -20.571632385253906, 29.64822006225586, 22.34722137451172, -18.74590492248535, 39.425540924072266, 19.822568893432617, 6.89939022064209, 16.405241012573242, -20.76827621459961, 50.191707611083984, 3.6866378784179688, 16.878461837768555, -3.5540645122528076, 27.067955017089844, 7.753046989440918, -17.766494750976562, 2.8550281524658203, -33.788700103759766, 17.051992416381836, 17.708606719970703, 2.533724069595337, -29.30185890197754, -31.838747024536133, -20.310293197631836, 15.429948806762695, 22.013139724731445, 10.098121643066406, -7.93691349029541, -21.167068481445312, -25.134572982788086, 23.580251693725586, -0.9943766593933105, 20.060956954956055, -16.095043182373047, 24.138835906982422, -36.02027893066406, -23.177534103393555, 0.8948748111724854, 10.882609367370605, -45.94520950317383, 14.306777000427246, -85.88651275634766, -67.33197784423828, 0.03890037536621094, 10.208354949951172, -7.027316093444824, -29.186321258544922, 22.91119956970215, 10.882919311523438, 69.30829620361328, -35.268348693847656, -1.342254877090454, -10.464653015136719, -15.516780853271484, -39.98645782470703, 45.76980972290039, 4.471909523010254, -60.078758239746094, 37.869659423828125, -7.172408103942871, 16.084903717041016, -55.765071868896484, -0.14632585644721985, -25.17778205871582, 33.371803283691406, 18.59743309020996, -7.2254252433776855, 11.00071907043457, 15.581291198730469, -20.794599533081055, -4.261524677276611, 40.224246978759766, -23.497140884399414, -18.36111831665039, 47.02054977416992, -5.260571002960205, -2.083519697189331, 22.356435775756836, -14.44359016418457, 2.0928690433502197, -27.19504737854004, -20.834932327270508, -24.933256149291992, 8.592294692993164, -27.127710342407227, -32.198516845703125, -35.18110275268555, 46.726158142089844, -21.011615753173828, -14.610319137573242, 13.612393379211426, -10.910524368286133, 36.269412994384766, -4.675780296325684, -0.1945713460445404, -15.713775634765625, 20.042118072509766, -19.059213638305664, -14.171774864196777, -16.52225685119629, -5.509144306182861, -34.322608947753906, -15.855497360229492, 1.386920690536499, -32.8811149597168, 2.1096420288085938, -26.147218704223633, 15.742524147033691, 37.968135833740234, 46.42850112915039, 2.410004138946533, 6.907444953918457, -32.98479461669922, 12.624059677124023, -10.397642135620117, 22.653705596923828, -19.587093353271484, 3.1719226837158203, -13.71554946899414, -4.5317301750183105, 33.78938674926758, -46.0058479309082, 13.272246360778809, 9.868556022644043, 0.185638427734375, -1.4782439470291138, -41.48931121826172, -10.655801773071289, 3.5845229625701904, 25.448427200317383, 0.48081982135772705, -49.59307098388672, -7.101171493530273, 32.247886657714844, -54.53884506225586, 20.636850357055664, -33.239234924316406, -11.273222923278809, -7.455079555511475, 6.748245716094971, 1.5248099565505981, 45.47279739379883, 47.594364166259766, -39.23623275756836, 3.3815035820007324, 3.6939685344696045, 6.03877067565918, 26.729646682739258, -22.067617416381836, 26.462617874145508, -4.950175762176514, -4.164827346801758, 44.44887924194336, -17.03887939453125, 23.6158390045166, -40.72815704345703, -27.942724227905273, 15.821470260620117, 52.76461410522461, 12.725456237792969, 4.93238639831543, -31.14366912841797, 13.101937294006348, -1.1690714359283447, 12.876030921936035, 6.398554801940918, 13.559225082397461, 3.7819151878356934, -32.551795959472656, -9.271978378295898, -16.63873291015625, -31.436777114868164, 29.488506317138672, -55.63532638549805, -18.370121002197266, -7.185513496398926, -16.182270050048828, 10.1005277633667, 0.5317025780677795, -27.713226318359375, 24.383590698242188, 4.362196922302246, -23.26370620727539, 7.490972995758057, -7.648280143737793, -7.863985061645508, 39.880165100097656, -4.7010111808776855, -58.189823150634766, 20.012470245361328, -15.538851737976074, 16.485551834106445, 18.90296173095703, 6.139213562011719, 69.45516204833984, -3.889758825302124, -4.078364372253418, -28.90139389038086, -77.82308959960938, -14.457172393798828, 33.50759506225586, -22.38170051574707, 23.071842193603516, 30.44835662841797, -16.272907257080078, 26.899372100830078, -5.500135898590088, -41.73799133300781, -60.4129524230957, 41.6480598449707, 31.0462589263916, -76.02302551269531, -31.28550148010254, 53.38865661621094, -31.154003143310547, 3.8767828941345215, -18.168895721435547, -7.368661880493164, 19.638450622558594, 18.860801696777344, -49.65880584716797, 46.2348518371582, -5.707709789276123, -35.794368743896484, -2.556213617324829, 13.585933685302734, 29.582834243774414, -11.108678817749023, 4.271866321563721, -8.413652420043945, -30.79334259033203, 80.95872497558594, -1.5650073289871216, 11.114004135131836, -46.96942138671875, 21.14816665649414, 32.910301208496094, -4.617023944854736, -72.37289428710938, 54.71480941772461, 1.4864071607589722, -38.99409484863281, 16.966102600097656, -35.50470733642578, 28.299528121948242, -44.875728607177734, 18.881309509277344, -56.07434844970703, -5.412461757659912, -21.405820846557617, -23.41865348815918, -55.55437469482422, 14.125604629516602, 39.78445816040039, -1.9640249013900757, -23.58457374572754, -46.20063400268555, -17.0858154296875, -4.951848030090332, -28.415069580078125, 28.636323928833008, 6.80661678314209, -20.05524444580078, -5.3216352462768555, 52.882259368896484, 10.612506866455078, 37.13663101196289, 32.115516662597656, -14.990067481994629, 56.510902404785156, -25.276016235351562, 37.609519958496094, 40.080387115478516, -7.384983062744141, 6.346176624298096, -31.65730857849121, 22.515201568603516, 36.38885498046875, 10.34114933013916, -31.21324920654297, -6.711516380310059, 23.049055099487305, 10.07894229888916, 30.189407348632812, -20.36799430847168, 6.163565635681152, 21.28864860534668, -25.634313583374023, 3.3577656745910645, 6.882318496704102, 9.55341911315918, -18.71866798400879, 12.165082931518555, -34.58583450317383, -1.9529837369918823, 54.02699661254883, -55.845455169677734, 33.74776840209961, 39.371543884277344, -1.6954375505447388, -80.68922424316406, -42.67073059082031, 5.485649108886719, -46.88172149658203, -2.089141368865967, -9.280508995056152, -56.06352233886719, -14.946127891540527, 4.00089168548584, 27.241065979003906, 58.02117919921875, -34.353214263916016, -7.258342266082764, -6.516688346862793, 5.172804832458496, 38.36159133911133, 43.28450393676758, -27.748828887939453, 9.746492385864258, -50.283485412597656, 15.37093448638916, -12.582319259643555, -54.9706916809082, -3.145634412765503, -2.186056137084961, -28.643903732299805, 19.673789978027344, 15.238094329833984, 27.083274841308594, -19.23402976989746, 26.04693603515625, 23.593416213989258, 27.45058822631836, -21.184207916259766, 36.671043395996094, 32.52708053588867, 11.627246856689453, -24.017560958862305, 26.701534271240234, -21.77263069152832, 34.4508171081543, 47.63872528076172, 66.68119812011719, 15.22719669342041, -6.4045867919921875, 50.86589813232422, 31.568941116333008, -90.49552154541016, -31.11614990234375, 10.59390640258789, 4.563714981079102, -33.84858703613281, 13.08323860168457, 1.894106388092041, -39.97236633300781, -67.14425659179688, -69.33795928955078, 7.670769691467285, 0.7877089977264404, 18.339824676513672, 5.510325908660889, 13.42644214630127, -47.758888244628906, -12.841704368591309, 13.680365562438965, -17.12163734436035, -22.800079345703125, -18.272502899169922, 27.18067169189453, 2.772841691970825, -51.93067932128906, -4.838315486907959, -60.12569046020508, 3.2973203659057617, 2.9674627780914307, 13.780102729797363, -9.774465560913086, 44.610836029052734, 33.44211196899414, -45.90340805053711, 1.8790252208709717, -8.624530792236328, -7.545500755310059, -22.345245361328125, 48.44789123535156, 23.963420867919922, -2.229872465133667, 29.596742630004883, 48.42426300048828, -18.590314865112305, 71.7614517211914, -38.450714111328125, -42.382572174072266, 53.43100357055664, 77.08226776123047, 84.66944122314453, 17.83883285522461, 8.216885566711426, 20.031904220581055, 7.865179061889648, 0.12369132041931152, 24.137218475341797, -7.074886798858643, -27.4735050201416, 1.086647629737854, -18.15327262878418, -32.85608673095703, 40.11787796020508, -11.897835731506348, 12.363722801208496, 44.46866226196289, -21.84223747253418, -4.066596984863281, -4.460169792175293, 6.1543426513671875, -40.4212646484375, 38.81465530395508, 14.251611709594727, 9.192300796508789, -3.7778139114379883, 52.276615142822266, 29.5144100189209, -0.0580744743347168, 39.17993927001953, -37.93485641479492, -22.361194610595703, 2.136760950088501, -9.564119338989258, 17.626901626586914, 2.0650362968444824, -19.131465911865234, 14.796754837036133, -9.351198196411133, -21.37357521057129, 17.002622604370117, -71.48028564453125, -0.32010728120803833, 40.10321807861328, 33.98295974731445, 17.14599609375, -12.191509246826172, -4.396195888519287, -9.014466285705566, 10.112123489379883, 6.134379863739014, -27.566312789916992, -28.04096794128418, -60.60812759399414, 75.33381652832031, 14.314794540405273, -9.819503784179688, -39.868873596191406, 13.113384246826172, 0.5894895792007446, -11.015511512756348, -75.95282745361328, 9.14090347290039, 22.80292510986328, -4.065929889678955, -28.882091522216797, -7.524206638336182, -31.651451110839844, -24.495615005493164, -4.097474575042725, -9.56639289855957, 11.190473556518555, -34.601890563964844, 49.636024475097656, -30.336441040039062, 35.73399353027344, 32.06180953979492, -45.30712127685547, -32.61286163330078, 26.538061141967773, 7.988579273223877, -2.6195755004882812, -48.992244720458984, 35.75344467163086, -8.503037452697754, 18.262907028198242, 29.806289672851562, 12.418612480163574, -5.7075347900390625, -11.788311004638672, -37.182823181152344, 8.781078338623047, -11.457535743713379, -30.042133331298828, 24.628734588623047, -11.829205513000488, -0.9481129050254822, 78.21183776855469, 22.535921096801758, -52.290897369384766, 6.356052398681641, 26.161235809326172, 23.241573333740234, -8.526677131652832, -4.776301383972168, -9.86660099029541, -24.120418548583984, 24.007190704345703, -12.941505432128906, 48.2531623840332, 48.77586364746094, 55.8170051574707, 2.336007833480835, -12.76009750366211, -39.291648864746094, -27.56239128112793, -22.31059455871582, -12.887670516967773, 16.43509292602539, -11.836495399475098, 7.896298408508301, 11.946795463562012, -35.0693473815918, 15.116068840026855, -12.770197868347168, -16.964969635009766, -15.870320320129395, 11.457895278930664, -39.89020919799805, -34.93898010253906, -8.01249885559082, -8.108382225036621, -32.45326614379883, -0.8535646796226501, -22.632160186767578, -33.77299118041992, -38.72779083251953, 31.536527633666992, -18.85375213623047, -43.11477279663086, -45.16385269165039, 46.6672477722168, -1.1852316856384277, 0.38054442405700684, 15.295120239257812, -24.42875099182129, 35.15263366699219, 8.322705268859863, -14.915168762207031, 12.60192584991455, -17.875835418701172, 41.899940490722656, -34.20643997192383, 34.1234130859375, -10.377447128295898, 9.057358741760254, 34.728858947753906, -1.5657674074172974, 30.477134704589844, 47.408042907714844, -14.965126037597656, 39.44674301147461, 7.230165004730225, -28.8350830078125, 11.380151748657227, 4.794198036193848, 27.522903442382812, -23.333951950073242, 16.52621078491211, 37.04410934448242, 18.58000373840332, -46.374813079833984, -72.3758316040039, 22.795482635498047, 15.91015338897705, 42.22266387939453, -34.62040710449219, 7.680456638336182, 4.797912120819092, 7.232441425323486, -25.944149017333984, 20.80718994140625, 7.802406311035156, -10.428410530090332, 22.553699493408203, -9.467517852783203, 6.749471187591553, -81.51990509033203, -43.0275764465332, -57.97444534301758, 19.520416259765625, 1.1256659030914307, -19.65937614440918, 7.491408348083496, 5.891253471374512, -23.80110740661621, 21.38751983642578, -7.914983749389648, 14.622454643249512, 5.20734977722168, -54.14265441894531, -13.796236991882324, 49.540557861328125, 17.67322540283203, 11.966571807861328, 31.565534591674805, 32.759281158447266, -22.957870483398438, 76.92680358886719, -21.423419952392578, 12.839385986328125, 13.4462251663208, 17.370311737060547, -46.08544158935547, -27.63607406616211, -3.9448204040527344, -9.824079513549805, -58.69339370727539, 74.96450805664062, 22.433032989501953, -6.360285758972168, -34.428382873535156, -31.407014846801758, 2.227320671081543, -60.964168548583984, -1.7244346141815186, 35.203468322753906, 1.2541671991348267, -21.3244686126709, 23.10108757019043, 6.468533515930176, 3.111494541168213, 27.486469268798828, -30.982563018798828, -41.41702651977539, 50.745880126953125, -31.711803436279297, 40.13020706176758, 7.65641975402832, 25.906177520751953, -24.677074432373047, 20.857404708862305, -24.854124069213867, -13.738143920898438, -50.714202880859375, 4.133857727050781, -1.4600054025650024, -11.66455364227295, -69.34488677978516, 2.5969767570495605, -52.18868637084961, 41.54906463623047, 6.53190803527832, 2.752932548522949, -7.082426071166992, 43.40808868408203, 4.610058784484863, 20.52256202697754, -36.52654266357422, -2.286097764968872, 50.529052734375, -2.1833322048187256, -17.250347137451172, -15.081409454345703, 17.5364933013916, -10.87328815460205, -13.802796363830566, 0.5514728426933289, 9.681647300720215, -37.68323516845703, 6.114372730255127, 20.347888946533203, -43.67817687988281, 50.46693801879883, 25.761188507080078, -2.8376872539520264, -4.615156173706055, -51.46718215942383, -11.157456398010254, -81.44831085205078, -31.920368194580078, -70.3121566772461, 23.29370880126953, -30.824886322021484, 16.623245239257812, -41.53634262084961, -26.751760482788086, 26.958106994628906, 21.247600555419922, 15.648298263549805, 8.333149909973145, 9.78919506072998, 51.404998779296875, 2.840414524078369, -32.33555603027344, 6.820029258728027, 2.0078179836273193, -11.027633666992188, 15.136878967285156, -39.92491912841797, -12.490110397338867, -5.320437431335449, 27.18264389038086, 18.42958641052246, 15.217350006103516, -57.36527633666992, 42.1591911315918, -60.55529022216797, 28.179826736450195, -22.916061401367188, -39.792259216308594, 25.381437301635742, -10.729510307312012, -40.75554275512695, -14.265134811401367, -11.75897216796875, -1.0943094491958618, -23.625545501708984, 29.755937576293945, 76.55094146728516, -9.040480613708496, 45.57760238647461, -26.25748062133789, -10.871150970458984, 10.300069808959961, 13.220993995666504, 21.111465454101562, -10.310739517211914, 12.268230438232422, -15.908282279968262, 17.760530471801758, 1.7250828742980957, 13.809678077697754, -20.3588809967041, -8.086526870727539, 9.649334907531738, 5.4011335372924805, 17.746997833251953, -4.73649787902832, 8.31202220916748, -9.280852317810059, 29.012155532836914, 15.091592788696289, 23.01693344116211, -24.603164672851562, 19.57588768005371, -22.56159782409668, -37.87655258178711, 14.385563850402832, 11.487631797790527, -1.1261768341064453, 6.085522174835205, -4.7335429191589355, -14.355083465576172, -30.991670608520508, 20.35395050048828, -19.809282302856445, -2.5761778354644775, 22.80082130432129, 59.817047119140625, 56.37645721435547, -9.341257095336914, -52.90822219848633, 25.917272567749023, 22.635465621948242, -1.9532201290130615, 6.6110992431640625, 12.110478401184082, -32.50203323364258, -24.746076583862305, 65.57742309570312, 8.138164520263672, 31.905290603637695, -0.544434130191803, -19.46245002746582, 1.45721435546875, 18.586551666259766, -9.674392700195312, 24.441452026367188, 13.16954517364502, -45.526206970214844, 15.512063026428223, 5.354860782623291, -20.341453552246094, -11.802467346191406, -1.8271520137786865, 49.74456024169922, 43.159603118896484, -34.91987991333008, 17.95069122314453, -32.46614074707031, -30.838947296142578, -0.6649226546287537, -24.23094940185547, 32.49295425415039, 38.026649475097656, 11.500027656555176, 4.575333595275879, 24.261816024780273, -3.407773971557617, -5.024536609649658, 24.801485061645508, -42.68187713623047, 10.642465591430664, 3.1658711433410645, -19.686887741088867, -1.7334160804748535, -10.875170707702637, 1.8163813352584839, 28.414323806762695, -28.190696716308594, -22.526674270629883, -18.698060989379883, -2.3284711837768555, 39.8303108215332, 63.99645233154297, 7.802089214324951, -11.218988418579102, 14.070003509521484, 24.128387451171875, -34.448238372802734, -12.61816120147705, 0.894982099533081, 29.81223487854004, -26.251001358032227, 42.26274108886719, -36.56995391845703, -48.70220947265625, 11.885456085205078, 18.2152099609375, 26.35417366027832, 20.160980224609375, -12.381414413452148, 9.492304801940918, -0.27657005190849304, -30.000736236572266, 61.452537536621094, -15.773307800292969, 22.89789390563965, 13.536231994628906, -29.58348846435547, 12.258035659790039, -1.1947524547576904, 18.79744529724121, -6.32090950012207, 2.0513062477111816, -17.179468154907227, -15.154007911682129, -48.340675354003906, -6.8944268226623535, -24.76743507385254, 46.2591552734375, 23.308700561523438, 43.98218536376953, 48.55197525024414, 22.319820404052734, -43.69610595703125, 14.481263160705566, 7.053096294403076, 55.99429702758789, -43.47760772705078, -45.21807098388672, 6.931756973266602, 29.170536041259766, -31.904569625854492, -27.944461822509766, 17.61195945739746, -48.916954040527344, 8.138015747070312, 12.39292049407959, -27.662118911743164, 56.20314025878906, 3.524582624435425, 0.6086958646774292, 8.340668678283691, -13.73111343383789, -26.11119842529297, -32.243804931640625, -35.21953582763672, -41.6053466796875, -37.05057144165039, 6.622197151184082, -17.94580078125, 26.045913696289062, 30.824373245239258, 8.998047828674316, -11.177435874938965, -0.8372759819030762, -9.784619331359863, -0.4722084403038025, 10.764305114746094, 12.336453437805176, 21.396303176879883, -15.579442024230957, 4.240101337432861, 8.03185749053955, 0.6794736385345459, -47.87384796142578, 9.613140106201172, 0.2662326693534851, -48.416690826416016, -30.691129684448242, 2.2512118816375732, 32.453556060791016, 47.9957275390625, -84.64057159423828, -29.81624412536621, -17.458934783935547, -5.9479451179504395, -21.097013473510742, 44.45730209350586, -0.5850305557250977, -29.82261085510254, 62.516441345214844, -4.867016315460205, 40.80534362792969, -13.556196212768555, -10.57621955871582, 24.43315887451172, -71.90247344970703, 6.7565107345581055, -54.27922821044922, 9.336462020874023, -9.44986629486084, -14.91804027557373, -20.991729736328125, -8.888703346252441, -20.469202041625977, 3.001192331314087, -44.18483352661133, 23.661684036254883, -47.000404357910156, -0.6981139779090881, -2.8942394256591797, -0.5935248732566833, 35.69593048095703, 14.847700119018555, 54.59342575073242, -23.496492385864258, 19.738765716552734, 20.084911346435547, 41.586177825927734, -26.665664672851562, 30.746257781982422, 10.66755485534668, -3.616842269897461, 15.852027893066406, 27.13526725769043, -44.86418914794922, 22.07967758178711, 28.330280303955078, 27.19099235534668, 33.095863342285156, 61.52353286743164, 17.242895126342773, 67.12676239013672, -5.701005458831787, 24.563634872436523, 17.02022361755371, -2.412628173828125, 24.788372039794922, -36.43060302734375, -23.690448760986328, -4.501245021820068, 21.11697006225586, 17.13079071044922, -36.978538513183594, 41.122066497802734, -36.49399185180664, -3.8788344860076904, 31.418682098388672, 21.678857803344727, 42.89949417114258, 23.89301300048828, 15.42306900024414, 37.057586669921875, 12.073022842407227, 38.881431579589844, 36.60419845581055, -6.919612884521484, -24.82724380493164, -12.334135055541992, -8.772263526916504, 9.693652153015137, 0.2714241147041321, -12.868276596069336, -62.27517318725586, -0.12791669368743896, 19.328227996826172, 10.441774368286133, -26.659963607788086, -11.573710441589355, -7.049133777618408, -27.80794906616211, -7.63625431060791, 11.677107810974121, 10.691450119018555, 14.009231567382812, 24.4398193359375, -11.990550994873047, 25.236785888671875, -4.435300827026367, 1.0018184185028076, -16.201221466064453, 48.0862922668457, 21.373493194580078, -8.636967658996582, 24.23007583618164, 49.64224624633789, -37.05587387084961, 65.98746490478516, -24.586158752441406, 25.419355392456055, -57.91746139526367, -39.623077392578125, 67.4113998413086, -27.442949295043945, 7.948450088500977, 8.286410331726074, 86.98255157470703, 6.133687973022461, 27.001230239868164, -18.38066291809082, 37.269752502441406, 18.732954025268555, 14.659041404724121, 63.23014831542969, -20.189359664916992, -13.211195945739746, -12.536589622497559, -1.5943710803985596, 6.967616081237793, -17.246028900146484, 1.6731479167938232, -18.021686553955078, -15.457489967346191, -22.62416648864746, 12.384462356567383, 5.345943927764893, -73.14228820800781, -48.907535552978516, 9.09945011138916, -10.073746681213379, 58.8447151184082, -0.7045181393623352, -3.4950335025787354, 11.488628387451172, 11.129524230957031, 9.488783836364746, -11.234536170959473, -0.6502511501312256, -36.120941162109375, 43.838096618652344, 35.44554138183594, 1.5310908555984497, -49.768375396728516, -21.566776275634766, 47.05086898803711, 39.7578125, -32.930580139160156, -10.351763725280762, 25.884614944458008, -63.19910430908203, -1.3221321105957031, -33.925601959228516, -38.2194709777832, 30.85367202758789, -17.659770965576172, -5.862820625305176, 44.41267395019531, 72.76703643798828, -43.98493194580078, 33.28288650512695, -39.74162292480469, -37.596065521240234, 0.3825710415840149, 21.137409210205078, 23.28021812438965, -15.680392265319824, 8.322905540466309, -24.49068832397461, -21.754491806030273, -0.21427273750305176, 9.167092323303223, -43.80419158935547, -47.001678466796875, -23.742719650268555, -5.992537021636963, 45.11530303955078, 35.10673904418945, -10.969854354858398, -9.495298385620117, 29.08200454711914, 3.327481508255005, -2.166079044342041, 45.3263053894043, -16.684463500976562, 51.022098541259766, 59.07589340209961, 42.69212341308594, 45.22170639038086, -44.36485290527344, 6.583436012268066, -25.895578384399414, -3.34812068939209, -35.00983810424805, 7.39306640625, 28.00421142578125, -12.891758918762207, 11.26366901397705, -52.317161560058594, -45.577613830566406, -18.071331024169922, -22.438743591308594, 31.504365921020508, -1.9866482019424438, -10.725473403930664, -21.66229248046875, 25.251840591430664, -5.523612022399902, -18.082143783569336, 10.425333976745605, 5.044182777404785, 15.736781120300293, -45.03702926635742, 25.240922927856445, -20.440319061279297, 0.9890785813331604, -28.342252731323242, -51.01350402832031, -20.94991111755371, -9.34374713897705, -0.5383458733558655, -15.710916519165039, 33.278167724609375, 39.87868881225586, 32.90694046020508, 34.014373779296875, 18.18567657470703, -38.02531051635742, 10.89965534210205, 36.18647766113281, -7.025007247924805, 40.14323806762695, 9.038948059082031, 9.057218551635742, 28.082782745361328, -58.70465850830078, -46.25605392456055, 7.928689956665039, -41.384056091308594, 34.53520584106445, 4.070481777191162, -52.680721282958984, 44.043582916259766, -6.140997409820557, 15.904569625854492, 46.16923904418945, -8.312528610229492, 39.19647979736328, -15.107230186462402, 19.84602928161621, 16.358915328979492, 44.426265716552734, 14.694188117980957, -18.7790470123291, 52.633827209472656, 30.424179077148438, 10.082865715026855, 39.21830368041992, 56.03502655029297, -30.214038848876953, 0.5544455051422119, -29.887243270874023, -32.617164611816406, -15.197501182556152, -2.894775629043579, 19.50018882751465, -21.18338394165039, 41.61747360229492, 13.021931648254395, 40.13967514038086, -23.31615447998047, -5.020459175109863, -3.0171234607696533, -13.00934886932373, 36.59271240234375, -13.259626388549805, 67.57272338867188, -8.091418266296387, 4.535005569458008, -19.141172409057617, -21.095014572143555, 63.11383056640625, -37.57614517211914, 1.1081699132919312, 13.737777709960938, -29.552127838134766, -19.422639846801758, -7.487085819244385, -25.21800994873047, -21.011991500854492, -4.869021892547607, 50.71036911010742, -11.28178596496582, 40.101863861083984, -23.281024932861328, 12.606746673583984, -27.45091438293457, 4.435103416442871, 31.83266830444336, 22.23833465576172, 7.4843058586120605, -55.7408332824707, -13.107461929321289, 12.319042205810547, -46.28791046142578, -19.798152923583984, -33.10469436645508, -19.641361236572266, -15.238762855529785, 19.513639450073242, 45.722530364990234, -9.088210105895996, -22.765878677368164, 38.36679458618164, 5.034830093383789, -74.18277740478516, 8.81303596496582, -7.007309436798096, 50.15772247314453, -42.67826461791992, -3.7819600105285645, 3.1470425128936768, -22.24290657043457, -4.989974498748779, -3.496831178665161, -5.081156253814697, 26.489742279052734, -22.314800262451172, 55.80172348022461, 21.28900909423828, -12.10077953338623, -68.75360107421875, -13.978671073913574, -55.7398681640625, -2.956974983215332, 2.778317451477051, 23.370153427124023, -22.744247436523438, 4.608728408813477, -24.394250869750977, -5.387650012969971, -6.844327926635742, -6.266198635101318, 25.04319190979004, 5.591931343078613, -2.083587884902954, 13.648785591125488, -3.7695999145507812, 38.83409118652344, -36.4000129699707, -0.8298267126083374, 21.04697608947754, 14.886241912841797, 10.157899856567383, 25.78490447998047, 8.595431327819824, 39.246009826660156, 34.5828971862793, 25.166231155395508, 39.08478546142578, -57.18498229980469, 16.02876091003418, 38.73051834106445, 7.929004669189453, -13.065958976745605, -61.717864990234375, 4.3165283203125, -2.556424856185913, 24.487014770507812, 29.850522994995117, -9.630356788635254, 67.32032012939453, 18.623058319091797, -0.7494636178016663, -29.479007720947266, 5.451729774475098, 8.482481002807617, -11.967987060546875, -23.613962173461914, 40.45158386230469, -62.867515563964844, 16.865219116210938, 60.873878479003906, -17.527454376220703, -0.6979281902313232, -43.398231506347656, 5.538311958312988, -8.942245483398438, -15.1466703414917, -15.8275728225708, -4.681544303894043, -7.138981342315674, -54.71248245239258, -34.94130325317383, -38.71565628051758, 22.27259635925293, 32.5212287902832, 18.527198791503906, -32.78645706176758, -20.6555118560791, 32.5631103515625, -67.66332244873047, 27.88938331604004, -36.90591812133789, 35.66337585449219, 17.465620040893555, 40.939762115478516, 5.850858688354492, 59.2199592590332, 25.058788299560547, -32.077674865722656, -52.01834487915039, -8.556142807006836, 8.812849044799805, 26.628023147583008, 30.562746047973633, 10.141908645629883, 50.995872497558594, -1.5737385749816895, 31.29827880859375, 39.2675666809082, 19.129133224487305, -19.129276275634766, 16.501405715942383, 16.788406372070312, -17.19051170349121, 9.686383247375488, 5.770127296447754, -15.777999877929688, 22.045936584472656, -15.332832336425781, -8.368919372558594, -15.322421073913574, -34.069068908691406, -3.525846004486084, 33.744590759277344, -60.069549560546875, -53.8531494140625, -58.102230072021484, -26.220956802368164, -26.3096866607666, -17.075054168701172, -66.72514343261719, 28.07290267944336, -10.143733024597168, -10.375913619995117, -14.003161430358887, 41.87345504760742, 11.943938255310059, 55.61172866821289, 4.523716449737549, -18.994056701660156, -22.873199462890625, -26.327341079711914, 15.533720016479492, -27.856605529785156, -13.454179763793945, -3.4691359996795654, -23.893613815307617, 3.5917255878448486, 42.082786560058594, -40.5800666809082, -27.454160690307617, 37.03528594970703, 9.739721298217773, -27.005552291870117, 27.952102661132812, -16.111968994140625, 30.86443328857422, -20.286975860595703, -38.97859573364258, -37.992488861083984, -52.726654052734375, 30.181427001953125, -48.78485107421875, -17.411270141601562, -3.2963740825653076, 0.4990781545639038, 9.791414260864258, -39.055885314941406, -47.9217529296875, -7.000448703765869, 32.0211067199707, 10.538081169128418, -23.633224487304688, -4.9912333488464355, 6.83519172668457, -26.92070960998535, 29.51922035217285, 52.68025207519531, 45.744049072265625, 6.9727067947387695, -3.185756206512451, -17.21421241760254, -36.20037078857422, -21.445507049560547, 61.89332962036133, 1.5891886949539185, -46.114933013916016, 49.6647834777832, 17.643701553344727, 0.6487337350845337, -30.49289321899414, 56.04779052734375, -17.867727279663086, 33.88052749633789, 16.441120147705078, -10.844867706298828, 6.175919532775879, -24.572105407714844, -34.81028366088867, 36.4401969909668, 31.079418182373047, 50.014869689941406, -25.715065002441406, 23.738130569458008, -12.75637435913086, 11.204008102416992, -35.54822540283203, 17.057584762573242, 0.5369998812675476, -34.71184539794922, -34.610965728759766, -0.9630950093269348, -44.465850830078125, -35.67655563354492, 40.87378692626953, -35.12600326538086], [7.121659278869629, 36.34189224243164, -0.8816482424736023, -46.120731353759766, -10.897615432739258, 28.861392974853516, -0.5076576471328735, 40.814571380615234, 11.958624839782715, -49.15812301635742, -25.44095802307129, -62.34652328491211, -8.453169822692871, -17.82560157775879, 13.345541954040527, -10.800440788269043, 5.0310564041137695, -38.14845275878906, -95.85488891601562, 5.864962100982666, -28.420137405395508, 42.54062271118164, 14.988709449768066, -20.193653106689453, -12.286600112915039, 14.085911750793457, 1.8363454341888428, -29.44333267211914, -37.976016998291016, -2.0096628665924072, -15.713218688964844, -17.727636337280273, -44.62328338623047, 34.836517333984375, -4.136429309844971, -10.075366973876953, 33.87120056152344, -17.47293472290039, 38.22983169555664, 37.4525260925293, -25.21219825744629, -2.465886116027832, 3.711026906967163, 14.869093894958496, -43.29437255859375, 25.123735427856445, 0.8379603624343872, 13.582830429077148, -14.812928199768066, -16.117456436157227, -14.73672103881836, -15.580323219299316, -15.825335502624512, -7.848635673522949, -43.5609245300293, -16.703683853149414, 6.32288122177124, -6.900771617889404, -26.07332420349121, 12.85031509399414, -17.96253204345703, 7.439847946166992, -24.796173095703125, -19.16925621032715, -17.174575805664062, 3.160836696624756, -4.385743618011475, 44.544715881347656, -33.95331573486328, 12.803464889526367, -10.863266944885254, 18.904821395874023, 3.558802604675293, -34.600154876708984, -20.215015411376953, 8.197418212890625, -66.25445556640625, 21.154733657836914, 0.5772669315338135, -46.866947174072266, -0.22900857031345367, -21.0618839263916, -49.257755279541016, -39.264034271240234, -27.907564163208008, -44.2730598449707, -10.5514497756958, -27.19699478149414, -8.97360610961914, -13.457216262817383, 22.413156509399414, 25.696409225463867, 2.1372063159942627, -12.408997535705566, -41.19923400878906, -27.83588981628418, -75.92613983154297, 19.51474952697754, 6.201204776763916, -21.90941047668457, -16.465030670166016, -0.11510296165943146, -6.935016632080078, 31.42706871032715, 8.012471199035645, -7.842944145202637, 20.807641983032227, 22.15846061706543, 15.708284378051758, 31.977432250976562, 23.621845245361328, -37.497108459472656, -4.484531402587891, -19.23436737060547, -3.984924793243408, 18.530105590820312, -15.494013786315918, 2.2149441242218018, -76.8153305053711, 46.18183898925781, -14.232828140258789, -21.689987182617188, 34.034912109375, 31.128726959228516, -20.53109359741211, 17.742982864379883, -31.336210250854492, -45.05931854248047, 17.192855834960938, 1.3636163473129272, -27.164384841918945, 48.290283203125, 15.041007995605469, 22.46425437927246, -42.99156951904297, 6.269021987915039, -78.95415496826172, 46.70648193359375, 35.415428161621094, 4.659329414367676, 17.35603904724121, 47.86517333984375, 3.1570324897766113, -42.63461685180664, 20.5297794342041, -36.09464645385742, 17.140850067138672, 9.063154220581055, 10.564053535461426, -14.73964786529541, -20.45985984802246, 5.066804885864258, -29.852073669433594, -14.411928176879883, -58.362117767333984, -24.739883422851562, 6.605786323547363, -15.067864418029785, 64.93407440185547, -51.4705810546875, 28.143081665039062, 87.72347259521484, 4.366737365722656, -4.064584732055664, 3.9250545501708984, -20.85319709777832, -24.798526763916016, 54.066444396972656, -49.86643600463867, -7.749028205871582, -20.720739364624023, 54.28824996948242, -0.1812494397163391, 25.646821975708008, 22.31037712097168, -4.637630939483643, -13.235633850097656, -5.583580493927002, -28.486406326293945, -10.7577486038208, -0.36499375104904175, -40.884254455566406, 25.772018432617188, 3.8516950607299805, -30.1961612701416, -7.315619468688965, 17.762859344482422, 13.850393295288086, 28.05937385559082, 37.917728424072266, 2.3248395919799805, 10.002542495727539, -7.530168533325195, 28.46420669555664, -34.85991668701172, 15.419723510742188, -5.551019191741943, -30.902177810668945, 11.036327362060547, -10.317167282104492, 9.300708770751953, 5.653420925140381, -37.56317138671875, 2.9821107387542725, -0.7881976366043091, 51.80442810058594, 8.055355072021484, -21.149864196777344, 9.324590682983398, -16.261474609375, -1.4977781772613525, 50.42965316772461, 103.07244873046875, 0.5926640033721924, -42.273353576660156, -12.883512496948242, 30.109058380126953, -33.456668853759766, 25.425569534301758, -24.337045669555664, -34.10833740234375, 1.0380192995071411, 15.705550193786621, 44.62621307373047, -35.220367431640625, -10.983716011047363, -22.716203689575195, 18.96582794189453, 28.69798469543457, -5.404872894287109, -30.565776824951172, 21.07436180114746, 5.91120719909668, -0.013624191284179688, -23.41778564453125, -0.8880232572555542, 40.61125946044922, -13.55884838104248, -3.711331844329834, -19.094934463500977, 4.0039520263671875, -0.6588305234909058, -38.20626449584961, -83.12779235839844, 36.59840774536133, -28.771726608276367, -33.07197952270508, 39.57700729370117, -3.500812530517578, 31.68750762939453, 5.057971000671387, -22.02358055114746, -4.647802829742432, 23.372896194458008, -38.06977081298828, 27.47258186340332, 44.61145782470703, 29.594751358032227, 28.023359298706055, 38.33972930908203, 11.92811393737793, 46.34105682373047, -13.62570571899414, -23.975725173950195, 2.019944667816162, 38.110023498535156, 9.649116516113281, 5.843213081359863, 15.624505043029785, 18.99414825439453, 23.10283851623535, -21.022817611694336, -17.112058639526367, -42.787654876708984, -19.019472122192383, 4.788848876953125, -8.669469833374023, -8.645909309387207, 45.5548095703125, 2.122664213180542, -17.14742660522461, 32.68913269042969, 29.65697479248047, -36.945037841796875, 16.454267501831055, 6.788933753967285, 7.8788533210754395, 8.700891494750977, -30.137853622436523, -7.885887622833252, -39.981807708740234, 5.181190490722656, -15.633374214172363, -7.336304664611816, 11.49036693572998, -33.43513870239258, 74.93624877929688, 32.439693450927734, 19.193313598632812, -44.80097579956055, 17.153072357177734, 74.97962951660156, -58.730987548828125, 37.371822357177734, 9.857519149780273, 16.06749153137207, 57.46455001831055, -37.34194564819336, 21.002561569213867, -45.981414794921875, -4.886679649353027, -10.729188919067383, 42.2970085144043, -22.127389907836914, 32.73835372924805, -31.917333602905273, -43.87320327758789, -25.809677124023438, 13.079995155334473, 58.97396469116211, -7.9968461990356445, -35.65005874633789, -17.77439308166504, -9.004945755004883, 16.82345199584961, -41.496238708496094, -9.063283920288086, -10.47470760345459, -36.88465881347656, 29.817625045776367, 12.693422317504883, 16.772907257080078, -8.190231323242188, -1.069875717163086, 13.265935897827148, 2.6502583026885986, 27.45887565612793, 13.187798500061035, 44.12367248535156, 3.951838493347168, -21.989919662475586, -116.15203094482422, -48.62934875488281, -45.117408752441406, 9.334426879882812, -12.148277282714844, 11.152918815612793, 10.146143913269043, 28.972610473632812, 29.326913833618164, 0.6858065128326416, -5.897507667541504, 4.172501087188721, 28.554874420166016, -1.9083300828933716, 23.5874080657959, 42.02588653564453, -13.4717435836792, 42.91650390625, 33.37361526489258, -16.98586082458496, -6.56473445892334, -29.500778198242188, -52.2910041809082, 27.14588165283203, 4.954983234405518, 14.03775691986084, 15.836665153503418, 15.284564971923828, -31.38304901123047, 5.599902629852295, -1.8466877937316895, 2.030545711517334, 18.340707778930664, 9.784058570861816, -11.55795955657959, 1.122272253036499, 34.3666877746582, -2.099935531616211, 16.994972229003906, -7.3975348472595215, 16.292482376098633, -37.81209945678711, 35.50343322753906, -5.74215841293335, -5.122684001922607, -16.917266845703125, 5.336397171020508, -20.669273376464844, -43.073997497558594, 27.6983642578125, 25.307453155517578, -10.700105667114258, -47.28806686401367, 63.34564971923828, -6.5036396980285645, -14.376762390136719, -11.188529968261719, 1.3141045570373535, 66.69886779785156, 13.025965690612793, -8.548959732055664, 6.541641712188721, 4.605177879333496, -34.58474349975586, 13.514074325561523, 29.073448181152344, 59.10044860839844, -28.05314064025879, 16.733428955078125, 24.030593872070312, -1.944440245628357, -30.77633285522461, 37.54985809326172, 3.5820891857147217, 26.118057250976562, -21.524600982666016, 38.12380599975586, 29.197059631347656, -29.095258712768555, -20.55198097229004, -18.61301612854004, 20.57413101196289, -14.078770637512207, -42.505027770996094, -34.21405029296875, -9.578927040100098, 1.4661283493041992, 1.6258714199066162, 46.94093704223633, 51.922481536865234, -10.077937126159668, 5.566091537475586, 2.463712453842163, 14.030655860900879, 21.173553466796875, -1.7948219776153564, 61.080379486083984, 3.7988665103912354, -52.47624969482422, -13.60047435760498, 11.000625610351562, 14.874259948730469, -50.243595123291016, 21.42293930053711, 14.934389114379883, 4.0638532638549805, 17.931060791015625, 38.03913497924805, -8.194725036621094, 35.13581085205078, -15.527482032775879, 13.468377113342285, -17.653696060180664, 18.23115348815918, 8.71430492401123, 0.9801865220069885, -14.60071849822998, 60.09482192993164, -0.5164082050323486, -11.446920394897461, -46.08757019042969, 18.885526657104492, -0.10190033912658691, 56.50611877441406, 31.181514739990234, -40.080204010009766, 34.16825485229492, -33.65655517578125, 26.996932983398438, 0.8301255702972412, -0.22262835502624512, -17.713260650634766, 14.320330619812012, -36.22623825073242, -14.894072532653809, 21.907896041870117, -2.179781913757324, -36.9613037109375, 0.17720818519592285, -27.33092498779297, 32.81010437011719, -21.47319984436035, -11.291863441467285, 47.86448287963867, 0.4110833406448364, 7.1826043128967285, 7.204422473907471, 42.223785400390625, 2.4162237644195557, 7.333982467651367, 19.658729553222656, 22.755186080932617, 20.46341896057129, -47.72450637817383, -3.509262800216675, 28.031023025512695, 21.527801513671875, -9.498506546020508, 32.68427276611328, -26.292682647705078, 28.49408721923828, 23.792198181152344, -7.0090484619140625, 13.840545654296875, -51.45362854003906, -32.34041213989258, -2.5853960514068604, -31.031898498535156, 14.082793235778809, 59.04359817504883, -4.22160005569458, 37.150508880615234, -34.782432556152344, 12.55081844329834, 53.17253494262695, 72.06069946289062, -18.907032012939453, -49.22502517700195, -11.048779487609863, -4.537886619567871, 5.997970104217529, 25.769670486450195, 31.473615646362305, 23.852825164794922, -13.350411415100098, 64.51644134521484, -14.099912643432617, -17.541202545166016, -41.40427017211914, -4.43825101852417, -7.088934898376465, 32.28076934814453, -21.706392288208008, 18.086593627929688, 0.3369518518447876, 4.916276454925537, 2.976149559020996, 5.840178489685059, -13.445972442626953, -96.40496063232422, 14.573776245117188, 0.862449586391449, 1.6082156896591187, -19.27373695373535, 4.117015361785889, -23.0997371673584, 43.94670867919922, -7.675347805023193, -46.140377044677734, -14.566370964050293, -21.179065704345703, 43.66929626464844, -40.63570022583008, 6.980828762054443, 5.517580032348633, 24.86945152282715, 45.14485549926758, 6.877041339874268, 50.468143463134766, 23.02080535888672, -22.686546325683594, -27.716745376586914, 39.27257537841797, -5.896682262420654, -65.24246215820312, 33.13780212402344, 76.69306945800781, 10.026749610900879, 17.37775993347168, -26.305540084838867, 4.551145553588867, 23.961166381835938, 2.897420883178711, -23.931968688964844, -10.791949272155762, -42.34004592895508, 52.97369384765625, -33.391136169433594, -105.25346374511719, 1.7692487239837646, 14.2239351272583, -32.251487731933594, 30.33045196533203, -23.626728057861328, 41.49139404296875, 3.883446216583252, 48.14936447143555, 2.694767475128174, -28.221460342407227, -55.118377685546875, -21.774499893188477, 1.6572866439819336, -13.623180389404297, -50.305091857910156, -0.11880417168140411, 17.857458114624023, -36.57106399536133, -10.185698509216309, 13.988509178161621, -19.272552490234375, -16.97748374938965, 3.9993414878845215, -48.69955825805664, -30.994569778442383, -37.91103744506836, -0.7426261901855469, 6.630716323852539, 22.88335418701172, 15.427943229675293, -4.669023513793945, -15.016735076904297, -36.383419036865234, -61.820274353027344, 13.671339988708496, 48.26840591430664, -0.4886329174041748, -3.798769474029541, -28.40772819519043, -70.8652572631836, 10.26992416381836, -8.847707748413086, -2.252053737640381, 28.687471389770508, -18.746675491333008, -19.60171127319336, 22.36136817932129, 3.4340083599090576, 16.210845947265625, -21.823135375976562, -12.904565811157227, -39.37164306640625, -5.889925003051758, -15.643412590026855, -33.852413177490234, -58.95225524902344, -11.541155815124512, 7.159740447998047, 4.870083808898926, -13.96280288696289, -6.261415958404541, 19.23164939880371, 10.205422401428223, 15.088065147399902, 1.728323221206665, 50.01812744140625, -6.441736698150635, -1.2998778820037842, -18.558347702026367, 0.11522603034973145, -74.25202941894531, 22.47460174560547, -36.720069885253906, 86.11422729492188, -11.86587905883789, 24.532852172851562, -60.89142990112305, -18.11608123779297, -0.46282172203063965, -21.3139591217041, -1.1460260152816772, 40.66080093383789, -53.676727294921875, -18.04997444152832, 1.4030036926269531, 13.910717964172363, 32.415992736816406, 21.61405372619629, 43.434349060058594, -35.214073181152344, -28.361228942871094, 20.239213943481445, 27.168331146240234, -18.27217674255371, -30.084869384765625, 21.72077178955078, 2.317701816558838, -2.325502872467041, -33.27997589111328, -0.9209111928939819, -15.80650520324707, 38.61895751953125, 12.281031608581543, 2.4188380241394043, -1.4303056001663208, -18.0035457611084, -15.844660758972168, 33.765838623046875, 11.08765983581543, 18.79302978515625, 15.832176208496094, 1.6092296838760376, -56.507598876953125, -2.134587526321411, -28.252582550048828, 17.881582260131836, 2.885162353515625, 11.536673545837402, -2.8785433769226074, 24.199220657348633, 12.084233283996582, 19.884830474853516, -29.142467498779297, 4.039012432098389, -17.228025436401367, -1.109195351600647, -26.395526885986328, -27.3391056060791, -62.475013732910156, 29.06624412536621, 6.079801559448242, 10.131719589233398, -19.014827728271484, 22.805490493774414, -30.130477905273438, -14.981757164001465, 31.53373908996582, -38.238399505615234, 37.6470947265625, -62.205955505371094, 4.8382415771484375, 43.56110763549805, 27.91864013671875, 34.138484954833984, 21.654626846313477, 14.05462646484375, -6.192609786987305, -0.30236178636550903, 9.682981491088867, -36.34845733642578, -21.143878936767578, -22.602745056152344, -12.318382263183594, -21.03290367126465, -21.709121704101562, -30.48369598388672, -39.387699127197266, 7.8516364097595215, -37.602169036865234, 44.889747619628906, -15.720253944396973, -2.2506284713745117, -11.804773330688477, 1.2021077871322632, -64.96692657470703, -29.945955276489258, 24.52275848388672, -7.969635963439941, -72.70307922363281, -9.704029083251953, -25.943065643310547, 12.815991401672363, 20.954681396484375, -35.48310470581055, 2.0191173553466797, -52.51478576660156, 37.74049377441406, -9.451805114746094, 33.32766342163086, 41.46260452270508, -15.604458808898926, -44.966495513916016, 13.616734504699707, 71.31722259521484, 39.483436584472656, -20.14900016784668, -7.946913719177246, -10.801990509033203, 11.114234924316406, 23.13423728942871, 4.080193519592285, 5.992623329162598, 40.51647186279297, -18.637516021728516, -16.48845863342285, -2.358205556869507, 7.2045745849609375, -36.004947662353516, 3.2214362621307373, -6.1467156410217285, -12.755167007446289, -48.850284576416016, -30.67224884033203, -19.961563110351562, 28.78794288635254, -63.3214111328125, 15.853863716125488, -4.738816261291504, -34.085506439208984, 9.176976203918457, -6.675314903259277, 44.030208587646484, 16.9824161529541, 27.591440200805664, -12.775012016296387, -16.289337158203125, 6.065221786499023, -38.2449836730957, -18.728731155395508, -4.433844089508057, 31.13811492919922, -25.700176239013672, 10.72166919708252, -9.383950233459473, -31.381534576416016, 42.12123489379883, 3.6422300338745117, 14.51916790008545, -22.734416961669922, -58.59788131713867, -7.6375322341918945, 16.919652938842773, 8.312726974487305, 41.48886489868164, -28.649324417114258, 15.310985565185547, 18.019733428955078, -47.425899505615234, 7.7619733810424805, -2.2073750495910645, 9.09254264831543, -24.34813117980957, 3.2396833896636963, 35.848960876464844, 0.04913508892059326, -5.8786396980285645, 14.515888214111328, 11.345958709716797, -82.22402954101562, 47.16935729980469, -45.30043029785156, -33.36713790893555, 29.104698181152344, 7.021839141845703, -46.57754135131836, -2.667022228240967, -27.88681983947754, -3.520040988922119, -23.50580406188965, 14.948774337768555, -1.7563077211380005, -5.7382426261901855, -49.88566970825195, -11.18712329864502, 14.892911911010742, 25.485881805419922, -22.145414352416992, 33.67163848876953, 15.696980476379395, 52.57072830200195, -18.865325927734375, -13.717313766479492, -16.314416885375977, -6.057744979858398, 8.209972381591797, 31.50298500061035, -11.025634765625, -11.392254829406738, -13.972681045532227, 34.44697189331055, -28.370891571044922, 18.389301300048828, 28.062654495239258, 5.936314582824707, -24.479312896728516, -26.116609573364258, -10.92603874206543, 22.807376861572266, -20.11176300048828, 39.52507019042969, -39.293922424316406, 16.060955047607422, 7.979733943939209, 17.334632873535156, 27.96563720703125, -30.311073303222656, 12.096722602844238, -22.42382049560547, 3.694737434387207, 9.547839164733887, 37.130699157714844, -11.005438804626465, -18.131912231445312, 10.860157012939453, 46.199562072753906, -32.08008575439453, -10.761674880981445, 13.630562782287598, 13.217368125915527, 8.700522422790527, -9.781631469726562, -1.4635183811187744, 25.36346435546875, 6.9977593421936035, -14.937732696533203, -9.730509757995605, 40.64533996582031, 31.672304153442383, -20.857398986816406, 30.175899505615234, -22.99981689453125, 12.0568265914917, -35.75080871582031, -11.166556358337402, 16.282896041870117, -63.699974060058594, 57.668678283691406, -11.440123558044434, 40.004005432128906, 7.6999006271362305, -11.966156959533691, 43.86582565307617, 52.78541564941406, -0.26926302909851074, 33.216468811035156, 29.848512649536133, 14.43317699432373, 18.94462776184082, -6.312620639801025, -10.35033893585205, 2.019397258758545, -52.7722053527832, -19.339277267456055, 30.53897476196289, 62.961669921875, 12.822477340698242, -9.20923900604248, 8.776952743530273, 16.94756507873535, 5.869413375854492, 0.966117262840271, 15.958532333374023, 15.605167388916016, -3.0482969284057617, -16.077098846435547, 14.289494514465332, -30.898942947387695, 22.44381332397461, 54.20524597167969, -23.958908081054688, 4.399016380310059, 42.25215148925781, 32.912757873535156, 3.517296075820923, -27.545988082885742, -32.906314849853516, 0.9089466333389282, -5.594368934631348, -36.30772018432617, -37.434085845947266, 7.009474277496338, -45.959617614746094, 44.90227127075195, -57.38638687133789, -28.80929946899414, -66.20420837402344, -3.961212396621704, 56.97059631347656, 72.14398193359375, -50.35554885864258, 22.989652633666992, -39.0505485534668, -11.853535652160645, 47.452430725097656, -7.224910736083984, -28.985261917114258, -9.520081520080566, -4.846197128295898, 10.467535018920898, 30.28278923034668, -60.43433380126953, 22.923982620239258, 20.17647933959961, -18.557212829589844, 60.41615676879883, -6.92579460144043, 6.602477550506592, 51.001365661621094, 40.68547439575195, -92.32620239257812, -2.894425868988037, 1.24873685836792, -0.8771748542785645, -12.378679275512695, -0.77171391248703, 19.376062393188477, -42.00965881347656, 17.34178352355957, -48.75762176513672, -54.52642822265625, -24.540102005004883, 43.83770751953125, -8.691372871398926, 21.4884090423584, 28.288114547729492, 7.851531982421875, 49.5466194152832, -39.41557312011719, 43.555908203125, 5.918806076049805, -27.68022346496582, -26.913713455200195, 14.932600975036621, -17.329280853271484, -53.83100128173828, 42.532230377197266, 0.06796827912330627, -89.7209243774414, -12.932767868041992, -14.747507095336914, 4.532455921173096, 12.926583290100098, 32.05069351196289, 50.35293197631836, -0.0067975930869579315, 38.918216705322266, -8.2617769241333, -10.230266571044922, -54.71356201171875, 4.1663994789123535, 43.15873718261719, -57.68434524536133, -34.819026947021484, 39.740562438964844, 21.795543670654297, -19.007278442382812, -42.72801971435547, 12.855902671813965, 13.935647964477539, -19.371580123901367, -12.975842475891113, 3.3390579223632812, -2.1183485984802246, -15.345216751098633, -26.20294189453125, 33.09246826171875, 45.68090057373047, -7.652305603027344, 47.435176849365234, -25.92568588256836, 17.623340606689453, 10.240978240966797, 49.5800666809082, 33.15715789794922, 11.706395149230957, 34.59387969970703, 5.250583648681641, 58.1993408203125, 10.066993713378906, -20.9143009185791, -3.103585720062256, 69.24356842041016, -48.885868072509766, 6.531246662139893, -3.361586570739746, 44.09030532836914, 29.753334045410156, -36.72673797607422, 3.0544612407684326, 45.28386688232422, -43.678810119628906, 1.3753020763397217, 19.339170455932617, 8.814838409423828, 41.824893951416016, -36.44793701171875, 22.075876235961914, -11.170016288757324, -16.55256462097168, 32.39673614501953, 16.41543197631836, -20.6137752532959, 49.18937301635742, 3.292511224746704, -1.9808974266052246, 16.42674446105957, 29.346630096435547, -77.78726959228516, 10.585416793823242, -28.572845458984375, 20.448246002197266, 15.280694961547852, 13.212563514709473, 6.501079559326172, -57.60315704345703, -25.229129791259766, 30.806785583496094, -55.523597717285156, 21.12380027770996, 2.5093929767608643, -20.884130477905273, -5.065424919128418, 34.04444885253906, 3.9153037071228027, -18.289030075073242, 0.35183000564575195, -16.447134017944336, 56.04826354980469, -1.5434054136276245, -60.18426513671875, 3.2406201362609863, -21.68798065185547, -21.51138687133789, -9.792224884033203, 28.942934036254883, -59.18147277832031, 25.56876564025879, -23.203767776489258, -34.19095993041992, 8.634906768798828, 8.208678245544434, -11.263641357421875, -3.051833152770996, 39.73564910888672, 0.9611785411834717, -9.606287002563477, -9.835746765136719, -33.049293518066406, 22.558412551879883, 26.270471572875977, 9.898606300354004, 5.942907333374023, -24.19740104675293, 6.729783058166504, 7.775968074798584, 2.516895294189453, -32.37540054321289, 6.81661319732666, -16.228445053100586, 42.73952102661133, 21.012969970703125, 27.24054718017578, -42.546173095703125, -8.207738876342773, 4.839841842651367, 4.79925537109375, -2.802670955657959, -45.643218994140625, 15.304387092590332, -24.32368278503418, 0.42643025517463684, -41.32057571411133, 38.287025451660156, 50.41762161254883, -38.28055953979492, -19.239364624023438, -11.591588020324707, 0.9040282964706421, -28.109487533569336, 37.646446228027344, 39.85052490234375, 14.420100212097168, -65.47726440429688, 11.171765327453613, 46.421875, -3.377441883087158, 67.89726257324219, -20.003286361694336, -60.99911117553711, -43.62416076660156, 18.96885108947754, -35.534698486328125, -42.703975677490234, -5.217033386230469, -4.730216979980469, -6.458057403564453, 3.2415528297424316, -14.161872863769531, 6.014176368713379, 34.88185501098633, 2.0246028900146484, -0.10305759310722351, -2.147252321243286, 39.151206970214844, -3.1311326026916504, 11.421828269958496, 38.04833984375, -38.53718566894531, -28.011138916015625, -17.387115478515625, -50.26579284667969, -13.901548385620117, 61.91542434692383, -18.809640884399414, 4.222780227661133, -29.26477813720703, 31.558792114257812, -33.02558898925781, -24.751535415649414, -24.560819625854492, 21.38461685180664, 22.95808219909668, 7.68606424331665, -4.7944865226745605, -25.861520767211914, -8.132862091064453, 15.635501861572266, 65.90184020996094, 21.645645141601562, -31.477651596069336, 31.935604095458984, -35.56373596191406, -57.963077545166016, -18.44964599609375, -3.5971546173095703, -2.5704219341278076, -9.118450164794922, 12.113545417785645, -3.8128738403320312, -48.165592193603516, -5.052496910095215, -22.318742752075195, -20.23834991455078, -35.71818923950195, 7.613551139831543, -2.2614011764526367, 19.252376556396484, 31.955995559692383, -7.459043979644775, 6.432290077209473, -40.47243118286133, 3.8510971069335938, -50.803550720214844, -15.567915916442871, -13.650918006896973, -26.473947525024414, 31.861478805541992, -56.74951934814453, -18.364870071411133, 38.585872650146484, 7.118273735046387, -23.69239616394043, 8.41529369354248, 29.087858200073242, 21.491300582885742, 19.66635513305664, 58.41798782348633, -6.201798439025879, 4.858645439147949, -0.6075732707977295, -8.315366744995117, -65.26739501953125, -8.801483154296875, -4.152879238128662, -11.657074928283691, 17.029151916503906, -17.378948211669922, 7.58860969543457, -50.89583969116211, 4.314065933227539, -32.34726333618164, 4.239192962646484, 11.770646095275879, -69.88407135009766, -47.79434585571289, 20.195165634155273, -12.087050437927246, -11.91010856628418, -31.435245513916016, 12.281238555908203, -33.53728485107422, 12.784790992736816, 12.054524421691895, -52.546207427978516, -9.942096710205078, 5.5262346267700195, -14.00192928314209, 1.6479220390319824, 8.401145935058594, -19.911453247070312, -9.663056373596191, -6.677940845489502, -0.45687389373779297, -62.59479904174805, -5.510062217712402, 21.331281661987305, 19.525222778320312, -0.17519760131835938, -24.10500717163086, -7.7042436599731445, -31.32683563232422, 15.99028205871582, -40.57423782348633, 3.0273072719573975, 13.919508934020996, 4.526674270629883, -2.803971767425537, 3.5842809677124023, 19.063188552856445, -50.999786376953125, 0.4157584309577942, 9.788751602172852, 28.500219345092773, 85.03108215332031, -15.605253219604492, -61.60302734375, 13.46241569519043, 0.9203540682792664, 55.235897064208984, 21.2271728515625, 39.372257232666016, 11.889330863952637, 25.933984756469727, 17.56051254272461, 5.202147960662842, -30.096946716308594, 44.10448455810547, 2.6410820484161377, -18.836624145507812, -8.040701866149902, -76.53620147705078, -3.484295129776001, 5.508725166320801, 99.85964965820312, 10.569059371948242, -2.6933131217956543, 40.553489685058594, -35.7071418762207, 23.092012405395508, -60.12244415283203, -72.0126724243164, -11.376659393310547, -1.217651605606079, 41.92637252807617, 15.221893310546875, -25.697914123535156, -0.5108414888381958, -54.14931106567383, 4.044771194458008, -26.687904357910156, 9.334726333618164, 22.999845504760742, 41.632911682128906, 4.684417247772217, -63.37924575805664, 8.341150283813477, -0.27599215507507324, 19.333158493041992, 47.66421890258789, -8.242679595947266, -18.2648868560791, -44.407691955566406, 43.4496955871582, -80.70655059814453, -9.691752433776855, -20.47804832458496, 3.733971118927002, -11.345146179199219, -10.165569305419922, 8.465259552001953, 23.25352668762207, -18.910917282104492, -24.972856521606445, 10.411917686462402, -0.015391290187835693, 16.23225212097168, 46.748661041259766, 74.28541564941406, -24.610185623168945, 9.196945190429688, 15.770047187805176, -32.491275787353516, -16.00925064086914, -32.77595520019531, 39.74456024169922, 18.85310173034668, 24.46682357788086, 3.7229485511779785, 29.181955337524414, 8.358309745788574, -13.241536140441895, 14.116877555847168, 18.871606826782227, 9.774928092956543, -32.8636360168457, 89.99860382080078, 21.08024787902832, 38.130775451660156, -13.068994522094727, 44.151710510253906, 12.77155876159668, -38.32855224609375, -4.788573741912842, 0.8807780146598816, -23.569801330566406, 29.49066162109375, -33.46502685546875, 21.10694122314453, -1.322208285331726, 8.317720413208008, -40.83546447753906, 32.017189025878906, 12.526848793029785, -50.165889739990234, -22.29079246520996, 21.178882598876953, -4.200828552246094, 9.688145637512207, 31.613601684570312, -13.08320426940918, 28.04791831970215, 35.19617462158203, 19.286706924438477, -46.32094955444336, 5.898815631866455, -64.73583984375, -7.813320159912109, 3.7763326168060303, 34.32801818847656, 1.770421028137207, -18.964345932006836, -6.43656587600708, 3.256692886352539, -9.161651611328125, 12.149179458618164, -16.547571182250977, 8.80312442779541, -9.011421203613281, 44.00970458984375, 3.5067780017852783, -9.885591506958008, -56.708900451660156, 3.7254536151885986, 35.91945266723633, -44.99386215209961, 29.627290725708008, 5.750080108642578, 10.90066909790039, -34.481658935546875, -2.950329542160034, -37.912757873535156, -11.2147798538208, -10.751715660095215, 66.8393325805664, -23.807384490966797, -43.39839553833008, -33.267215728759766, 12.800458908081055, 34.375335693359375, 24.54865074157715, -8.901228904724121, 2.1493232250213623, -10.26172924041748, -23.49309539794922, -18.799196243286133, 49.75165939331055, -11.973134994506836, -16.913583755493164, 69.95640563964844, -39.997535705566406, -31.060266494750977, -41.41252517700195, 22.063610076904297, 7.755459785461426, 61.63236999511719, 10.349571228027344, 60.21648025512695, -34.436649322509766, 35.23781967163086, -35.85394287109375, -3.0441997051239014, -24.70450210571289, 21.87068748474121, 25.277162551879883, 16.518449783325195, 26.55990982055664, -42.38548278808594, 60.761077880859375, -33.859046936035156, 2.689919948577881, -10.214388847351074, 49.23882293701172, 2.1711058616638184, -54.001243591308594, -3.966721534729004, 25.745996475219727, -20.376310348510742, -12.422982215881348, -10.119301795959473, -9.06197452545166, 9.216203689575195, 37.954349517822266, 8.908239364624023, -1.0363997220993042, -38.763240814208984, -15.815144538879395, 39.10939025878906, 19.039091110229492, -24.793655395507812, 26.99618911743164, 7.526274681091309, 41.55241012573242, 4.171876907348633, 30.93661117553711, -32.48841857910156, 20.231367111206055, 16.318498611450195, 11.24212646484375, 37.54612350463867, -39.28602981567383, 29.841493606567383, -10.420662879943848, 14.064713478088379, 72.17965698242188, -5.222769737243652, -22.8737735748291, 34.28874206542969, -59.088565826416016, -21.33845329284668, 19.54399871826172, 116.95682525634766, -42.626373291015625, 2.8665575981140137, -34.395294189453125, 33.97053909301758, 12.189041137695312, -15.411172866821289, -28.498689651489258, 12.511374473571777, 16.197032928466797, -11.416900634765625, 31.34518814086914, -17.932369232177734, 10.660089492797852, 43.64232635498047, 34.00542068481445, -7.484429836273193, 53.370113372802734, 47.27861785888672, 0.32809722423553467, -57.90605926513672, -37.535240173339844, 5.011539459228516, -13.848345756530762, 10.951565742492676, -2.397588014602661, -43.46866226196289, -25.140716552734375, -11.439865112304688, -17.34029197692871, -29.000871658325195, -46.74586868286133, 13.038639068603516, -39.637569427490234, 61.2911262512207, 68.2815170288086, 32.894351959228516, -28.321142196655273, -13.382471084594727, -24.623144149780273, -0.830335259437561, 34.879058837890625, -19.144027709960938, 45.66059494018555, -15.588342666625977, 3.9408140182495117, -34.777462005615234, 10.078842163085938, -8.471903800964355, -43.3563117980957, 10.838281631469727, 19.755165100097656, -20.677709579467773, -21.207252502441406, 51.40181350708008, -17.352415084838867, -13.068164825439453, 4.383932590484619, 24.16289710998535, -3.320608377456665, -32.943206787109375, -14.13027286529541, 33.52079772949219, 17.684846878051758, -8.450103759765625, 52.94151306152344, 1.1897437572479248, 4.735645294189453, -30.592090606689453, -36.3695068359375, 26.736915588378906, 17.219728469848633, -40.277217864990234, 31.392589569091797, 42.357662200927734, -0.6231404542922974, 45.03302001953125, 3.350844383239746, 11.792000770568848, -13.858502388000488, -7.223826885223389, 36.852352142333984, 33.689884185791016, 10.72056770324707, -36.442970275878906, 13.909008026123047, -22.058595657348633, 46.40739059448242, -9.674844741821289, 15.606112480163574, 4.793919086456299, 37.02642059326172, -72.48430633544922, 6.435583114624023, -32.02727508544922, 7.868551731109619, 48.83191680908203, -37.27706527709961, 26.441242218017578, -8.552671432495117, -14.349949836730957, 21.461496353149414, 5.5582122802734375, -5.086190700531006, 61.665897369384766, -32.62188720703125, -6.815368175506592, 107.60562133789062, -40.81462478637695, 22.719865798950195, -19.81202507019043, 3.5535268783569336, -8.264583587646484, 19.027538299560547, -11.258018493652344, -1.6573946475982666, 41.773529052734375, -35.175296783447266, 61.941192626953125, -16.36203384399414, -4.306868076324463, -12.328330993652344, 45.70661926269531, 21.57252311706543, -65.9347152709961, 16.17267608642578, -58.033531188964844, -16.660850524902344, 39.31105041503906, 4.718242168426514, 69.61126708984375, 72.91779327392578, -31.740772247314453, 9.722128868103027, -27.907503128051758, -13.524425506591797, -2.5651421546936035, 14.574185371398926, 35.16829299926758, 14.153681755065918, 25.35377311706543, -22.82160758972168, -17.62150001525879, -18.38855743408203, 31.579425811767578, 29.669513702392578, 27.597482681274414, 2.1554043292999268, -3.783513307571411, 7.777310848236084, -12.08002758026123, -19.271076202392578, 43.68159103393555, -6.294100761413574, 27.149368286132812, 6.462532997131348, 31.1632137298584, -13.429365158081055, -37.65613555908203, 35.02464294433594, -18.845998764038086, -35.116275787353516, -13.444204330444336, -24.004606246948242, 11.42009162902832, 26.0338134765625, -42.01272964477539, -9.43915843963623, -42.389503479003906, -58.364593505859375, 20.212491989135742, 77.68368530273438, 41.36271667480469, 12.084864616394043, -11.344970703125, 11.879105567932129, 12.400399208068848, 25.863361358642578, 1.7300670146942139, 1.1225930452346802, -13.866449356079102, 44.79714584350586, 27.96390724182129, 7.055230140686035, 35.68447494506836, 44.68265914916992, 20.708614349365234, 17.799846649169922, -26.090417861938477, 30.951040267944336, 2.1337890625, -11.816633224487305, -56.34673309326172, 21.622770309448242, -40.90338897705078, -49.83649444580078, 8.799391746520996, -45.627506256103516, -37.59238052368164, 8.030109405517578, -22.982913970947266, -29.955711364746094, -37.33378982543945, 24.2434024810791, -18.712387084960938, -34.1173095703125, 20.956571578979492, 4.994929790496826, -7.904978275299072, 30.278614044189453, 24.99087142944336, 1.0370107889175415, 9.641073226928711, 39.20916748046875, -3.5877504348754883, -26.182905197143555, 22.505977630615234, -6.190751075744629, 3.380540370941162, 6.5719451904296875, -21.438961029052734, 8.389750480651855, -12.139226913452148, -6.9772210121154785, 11.097794532775879, 16.94322967529297, -19.425790786743164, -25.543474197387695, 33.87394714355469, 24.44170570373535, 0.5852859020233154, -51.14002227783203, 18.760772705078125, -12.801554679870605, -26.25773811340332, 7.630276203155518, -80.8099594116211, 31.4893856048584, 27.400562286376953, 2.6703929901123047, 11.697895050048828, -26.392404556274414, -37.49361038208008, 3.133296489715576, 20.731597900390625, -19.76663589477539, -2.433736801147461, 36.30133056640625, 63.08741760253906, 39.62306213378906, 3.0161867141723633, 23.768497467041016, -9.824307441711426, 10.469515800476074, 40.115318298339844, 42.21571731567383, 5.540964603424072, 8.879754066467285, 5.879050254821777, -22.691314697265625, 16.836240768432617, -3.5114448070526123, -33.203800201416016, -50.495399475097656, 13.754837036132812, -24.529685974121094, -29.597166061401367, 37.91400146484375, 6.1512041091918945, 26.315105438232422, -21.01599884033203, 16.783655166625977, 29.786855697631836, -14.187732696533203, -0.4323679208755493, -35.713260650634766, -46.836341857910156, -8.704344749450684, -14.74604606628418, 35.7721061706543, 12.590380668640137, 35.69472122192383, 8.667168617248535, 14.787939071655273, 18.160146713256836, 13.262529373168945, -2.009624481201172, 19.724899291992188, -9.420024871826172, -29.247039794921875, 16.46746063232422, 6.482231140136719, -3.5298352241516113, -39.0861701965332, 43.81254959106445, -12.440855026245117, 9.971285820007324, 3.4057388305664062, -10.864075660705566, -20.9746150970459, 23.212406158447266, -40.420536041259766, -16.17022132873535, -61.26347732543945, 2.533545970916748, -31.548812866210938, -46.87980651855469, 13.058369636535645, -46.55430603027344, 12.668920516967773, -61.832679748535156, -6.702899932861328, -10.646937370300293, 3.2754602432250977, 17.369495391845703, 27.312271118164062, -45.416038513183594, -31.48672103881836, -46.107425689697266, -17.14354133605957, -0.729015588760376, 61.756195068359375, 9.985662460327148, 1.7438828945159912, -56.91530227661133, -25.33251190185547, 28.744184494018555, 31.158588409423828, 53.903316497802734, 8.37590217590332, -17.710643768310547, 19.209774017333984, -6.387546062469482, 34.848480224609375, -32.917850494384766, -0.059749312698841095, -17.34552574157715, -14.363906860351562, 3.7027173042297363, 9.244443893432617, -14.353918075561523, 38.95541000366211, -27.9331111907959, -19.884483337402344, -2.9008121490478516, 10.77341365814209, 5.347136974334717, 39.70240020751953, -22.489242553710938, -24.520368576049805, 29.142269134521484, 59.360679626464844, 79.74493408203125, 4.052847862243652, -10.165730476379395, -10.290698051452637, -3.1537961959838867, -23.5675048828125, -21.007835388183594, -13.354405403137207, -9.199531555175781, -8.325835227966309, -59.40545654296875, -58.18558120727539, -7.880773544311523, -24.98701286315918, -33.802711486816406, 19.775466918945312, -8.523468971252441, 12.077699661254883, -1.7074967622756958, -5.842813491821289, -16.903446197509766, 79.71051788330078, -44.37830352783203, -18.38811683654785, -70.560791015625, 39.627044677734375, -33.560752868652344, -38.70746612548828, 10.416171073913574, -46.31706237792969, 6.363774299621582, -9.303689956665039, -45.653961181640625, -22.922367095947266, 33.20452117919922, 28.482677459716797, -29.201766967773438, -44.776554107666016, 60.52750778198242, 22.851211547851562, 9.263668060302734, -30.69930648803711, -61.83382034301758, -3.2245442867279053, 3.4514083862304688, 31.4215087890625, 12.262449264526367, 14.2991304397583, 40.143741607666016, -45.718875885009766, 58.55655288696289, 13.554621696472168, 2.778486490249634, 16.069351196289062, -9.758852005004883, 10.290242195129395, 9.019599914550781, 13.573650360107422, -2.1100714206695557, -26.408601760864258, -48.93172073364258, 2.343430757522583, -38.92442321777344, -10.802145957946777, -23.261594772338867, -34.0953483581543, -13.95824146270752, 7.819687843322754, 18.041391372680664, 22.360389709472656, -51.531166076660156, -28.329801559448242, 1.4469640254974365, -35.054256439208984, -19.441221237182617, 35.08564376831055, -18.702787399291992, 17.388545989990234, -17.940980911254883, -40.129398345947266, -0.6467431783676147, -67.74534606933594, -14.588872909545898, 50.12958908081055, -7.630829811096191, -6.835317611694336, -1.511852502822876, -27.307558059692383, 29.68385124206543, 3.6529133319854736, -1.91300630569458, -32.826622009277344, -9.295442581176758, 50.69962692260742, -10.802584648132324, -27.364534378051758, 76.79747009277344, 3.234247922897339, 14.993987083435059, 20.222850799560547, -16.429195404052734, 40.989810943603516, -16.691654205322266, -8.381290435791016, 0.8737444877624512, 42.169952392578125, 3.3160719871520996, 24.499448776245117, 17.302135467529297, 27.188474655151367, -5.389304161071777, 34.58079528808594, 9.251602172851562, 11.98628044128418, -16.060489654541016, 29.050168991088867, 30.447551727294922, 8.170368194580078, 34.64206314086914, -7.224884986877441, -35.67430877685547, 35.54116439819336, 4.162436485290527, -38.80671691894531, -8.66433048248291, -28.518566131591797, 6.448959827423096, -27.623743057250977, -13.391752243041992, 48.52244567871094, 49.771453857421875, -6.386044025421143, -14.0431489944458, 13.721230506896973, 10.045768737792969, 2.5070509910583496, 64.91535949707031, -13.461224555969238, -2.2387290000915527, -14.541579246520996, 2.030660390853882, 17.264114379882812, 4.690061092376709, -0.678070068359375, -42.91800308227539, 1.9245175123214722, -36.40919876098633, 27.367565155029297, -32.33252716064453, -62.01786804199219, -1.0525293350219727, -0.5715653896331787, -12.08603286743164, 42.16141891479492, 32.15727996826172, -37.62287902832031, 29.92788314819336, -40.79847717285156, 5.12103796005249, 51.24113082885742, 20.872886657714844, -52.074546813964844, 19.126083374023438, -26.965900421142578, 11.131858825683594, 76.76049041748047, -56.19289779663086, 16.963233947753906, 0.3063289523124695, 26.178348541259766, 2.054074764251709, 44.367958068847656, 6.3488874435424805, 24.38809585571289, 17.962894439697266, 54.09547424316406, -23.45146942138672, -82.41300964355469, 2.2805166244506836, -14.502009391784668, -6.755841255187988, 23.93939971923828, -44.70911407470703, -19.493244171142578, -55.58076095581055, 18.888654708862305, -40.97285079956055, 14.428987503051758, -16.593013763427734, 45.691505432128906, -29.497283935546875, -0.7759740352630615, 19.457090377807617, -86.8100814819336, -7.015859127044678, -16.030094146728516, -45.57305145263672, 28.319250106811523, -25.34765625, -60.53190994262695, 9.353557586669922, -60.132389068603516, 11.482439041137695, 8.470672607421875, 6.055482864379883, -21.819440841674805, -23.734817504882812, -0.26335281133651733, -14.026638984680176, -30.564565658569336, -13.928799629211426, -55.80801010131836, 6.747028827667236, -6.107709884643555, -6.1100568771362305, -13.71187686920166, -30.087467193603516, 23.51256561279297, 1.309230089187622, -62.55197525024414, -24.284008026123047, 23.76236343383789, -47.2026481628418, -17.427810668945312, -68.34693145751953, -62.368438720703125, -40.26438903808594, -7.207674503326416, -10.960789680480957, 44.547088623046875, -64.4641342163086, -37.929683685302734, -31.143003463745117, -12.333303451538086, 19.647302627563477, -96.13994598388672, 9.871456146240234, -0.16488122940063477, -28.169435501098633, 14.477934837341309, -52.07548141479492, 41.202964782714844, 23.622251510620117, -16.053516387939453, -49.04649353027344, 58.722877502441406, 25.810903549194336, -2.9491066932678223, 26.291141510009766, 24.788955688476562, -27.142757415771484, -20.60025978088379, 5.761375427246094, -8.93300724029541, -27.486963272094727, 48.78947067260742, -9.31235122680664, -25.97593879699707, -1.2008347511291504, -63.41617965698242, -0.8435803651809692, 27.218040466308594, 44.37728500366211, 29.15727996826172, -0.16342854499816895, -0.621647298336029, 63.40260314941406, -7.117849349975586, 10.470922470092773, -12.47618579864502, 3.9010958671569824, 15.782835006713867, -14.139702796936035, -31.435270309448242, -45.19084167480469, -3.5027718544006348, 70.2492904663086, -3.084989070892334, -7.886791706085205, 41.525787353515625, 6.300878524780273, 18.117034912109375, -23.946754455566406, -42.86441421508789, 16.41257095336914, 53.59718322753906, -35.51414108276367, 8.383707046508789, 1.0548698902130127, 24.929466247558594, -27.699031829833984, 25.43779945373535, 20.89505386352539, 24.114282608032227, -21.464725494384766, -14.755892753601074, 10.025764465332031, 19.983217239379883, -26.156551361083984, -42.80647277832031, 34.438663482666016, 17.757488250732422, -30.462932586669922, -3.3814969062805176, 49.992652893066406, -4.340208530426025, 29.597972869873047, -39.07927322387695, -30.007076263427734, 5.494966983795166, -14.551427841186523, 26.049518585205078, 16.309694290161133, 5.8104376792907715, 2.0496115684509277, 37.39177322387695, 25.39068031311035, -17.457544326782227, -22.950529098510742, -36.320579528808594, 38.38267517089844, -10.798934936523438, -23.038345336914062, -19.75320816040039, -12.298566818237305, -67.3165283203125, 28.24510955810547, -4.102203369140625, -1.4999358654022217, 28.805583953857422, 16.070409774780273, -8.837129592895508, -33.768009185791016, -25.114057540893555, -28.118608474731445, 41.80816650390625, -11.187969207763672, -6.074303150177002, -20.980253219604492, -41.688594818115234, 34.03660202026367, -19.300643920898438, -23.788253784179688, -30.716346740722656, 11.68914794921875, -19.95792579650879, 6.873788833618164, 6.985246181488037, -24.925989151000977, -27.95842933654785, 6.768590450286865, 85.18905639648438, -32.59381866455078, 42.97919845581055, 14.575450897216797, -30.6690673828125, -17.902273178100586, 8.192781448364258, 3.4357666969299316, 0.4475952982902527, 26.2918701171875, -47.40898132324219, -24.039831161499023, 50.02153778076172, -24.316160202026367, 8.260269165039062, -58.11101531982422, -27.96390724182129, -23.68755340576172, -20.36161231994629, 0.6622717380523682, 21.97072410583496, 41.819580078125, 8.44921588897705, -10.088376998901367, 4.488541126251221, 2.995912790298462, 22.05791664123535, 13.901981353759766, 49.399375915527344, -9.372322082519531, 2.3774447441101074, 2.584050416946411, 2.7726128101348877, 6.921367645263672, -30.122896194458008, 0.8889892101287842, 28.69089126586914, 13.743468284606934, -15.04319953918457, 45.5697135925293, -41.15780258178711, -26.838359832763672, 59.527767181396484, 34.65348815917969, -41.3245964050293, 35.67461395263672, 19.405357360839844, -55.35158157348633, 34.51673126220703, 9.497233390808105, 4.973596572875977, 27.292226791381836, 74.67943572998047, -25.24386978149414, -36.6445426940918, -42.33469772338867, -14.422163963317871, 34.01670837402344, -17.390396118164062, 11.536211013793945, 31.034128189086914, -42.03144454956055, 11.77098560333252, -2.037623882293701, -38.42803955078125, -26.25262451171875, -5.670853614807129, -29.766685485839844, 1.2547378540039062, 0.1131364107131958, -20.76877212524414, -13.263862609863281, -13.744675636291504, -34.238685607910156, 42.88154602050781, 34.27340316772461, 24.87898063659668, 4.830014705657959, -37.871822357177734, 26.91033363342285, -72.72425079345703, -4.232797622680664, -20.20297622680664, 40.73983383178711, 6.809823036193848, 17.083999633789062, -0.05240166187286377, 37.158260345458984, 27.36492156982422, 29.695714950561523, -3.4729111194610596, 20.235952377319336, 33.88380432128906, -3.6810898780822754, 13.588096618652344, -25.976123809814453, -26.900806427001953, -28.1131649017334, -12.814988136291504, -1.1535019874572754, 0.02279496192932129, 42.71974563598633, -5.489959239959717, 53.43054962158203, -28.231704711914062, 37.18485641479492, 6.255326747894287, -23.525732040405273, -8.128891944885254, -39.42619323730469, 3.4716219902038574, 12.79500961303711, -5.572215557098389, 2.5635781288146973, 1.4557603597640991, -20.6496639251709, -27.470062255859375, 2.566493272781372, 33.49946212768555, -19.612791061401367, 14.14743423461914, -12.299080848693848, -7.284043788909912, -75.5118637084961, 10.541991233825684, 13.468766212463379, 34.59632873535156, -29.92040252685547, -26.968515396118164, -3.509838581085205, 16.955625534057617, -12.360482215881348, -26.298078536987305, 16.685123443603516, 12.537141799926758, 20.865270614624023, 15.20504093170166, 41.66152572631836, 11.641611099243164, 28.92320442199707, 0.8174546957015991, 23.940771102905273, 3.7239489555358887, 18.78067398071289, -14.665863990783691, 13.523306846618652, 26.215179443359375, -17.316259384155273, -34.44772720336914, 31.873891830444336, -9.329532623291016, 28.225540161132812, -12.148883819580078, 11.887439727783203, 17.324615478515625, -0.35684454441070557, -42.95357894897461, -20.927675247192383, 1.5161911249160767, -20.690793991088867, -38.625152587890625, 15.140985488891602, -5.771071910858154, 22.09244728088379, -67.5703125, 6.695101261138916, -37.87911605834961, 11.920767784118652, -4.20859956741333, -15.562846183776855, -43.91865921020508, 50.22492218017578, -19.00496482849121, -3.430147647857666, 9.510710716247559, -50.77587890625, -8.070613861083984, 24.997772216796875, 29.41813087463379, 21.955711364746094, 0.23100003600120544, 64.69222259521484, 28.93770980834961, 27.468332290649414, -15.441210746765137, -2.2667293548583984, 11.946516036987305, 66.02375793457031, -10.995331764221191, 26.413053512573242, -33.28351974487305, -41.9736213684082, 19.18988609313965, 30.9919490814209, -20.045307159423828, -7.837229251861572, 45.415077209472656, 20.828702926635742, -15.346455574035645, -75.61631774902344, 41.72569274902344, -26.320585250854492, 13.293261528015137, 11.34370231628418, -10.942046165466309, 30.399574279785156, 7.28640079498291, -19.761367797851562, -26.901103973388672, 27.513172149658203, -14.227848052978516, 13.213268280029297, -45.19728469848633, 17.876638412475586, -4.037783622741699, -7.27248477935791, 1.9824217557907104, -29.40751075744629, -19.18059730529785, 11.937560081481934, -18.007368087768555, 19.52287483215332, 24.926652908325195, -20.722280502319336, 5.023813724517822, 19.04926109313965, -9.445734977722168, -21.188276290893555, 10.80813217163086, 8.757247924804688, 21.308635711669922, -32.28729248046875, -11.046517372131348, 1.9451708793640137, 0.37246859073638916, 47.32590103149414, -27.606447219848633, -4.9601569175720215, 5.6591291427612305, -2.018582344055176, 15.548465728759766, 12.275435447692871, 9.136727333068848, 5.624774932861328, -0.07278990745544434, 37.80745315551758, 16.48339080810547, 14.222896575927734, -2.6178934574127197, 14.790620803833008, -44.036705017089844, -4.48814058303833, 6.520407199859619, 20.095081329345703, 11.84309196472168, -17.54543113708496, 37.16307067871094, -4.882133960723877, 12.637523651123047, 14.600948333740234, 3.614126205444336, -15.220132827758789, -57.14217758178711, 36.43692398071289, 101.63568115234375, 28.672487258911133, -32.76943588256836, 12.930013656616211, 1.0838483572006226, -19.47091293334961, 44.65458679199219, 34.020790100097656, -12.770263671875, 15.63531494140625, 30.68058204650879, -33.695369720458984, 33.675392150878906, 19.479568481445312, -3.3943591117858887, -28.066164016723633, -5.842773914337158, -47.581485748291016, 50.983558654785156, -30.51112174987793, -22.935016632080078, -11.265827178955078, 9.062028884887695, 82.57291412353516, -44.45269775390625, -45.39323806762695, 4.22941780090332, -57.72565460205078, 22.224451065063477, -2.0402252674102783, 14.118749618530273, 48.79310989379883, -22.38982582092285, -31.86646842956543, -11.556442260742188, -11.484148025512695, 11.895845413208008, -58.14039993286133, -8.978492736816406, 31.360477447509766, 3.394589900970459, 0.47207140922546387, -5.807890892028809, -11.48796272277832, -2.698646068572998, -16.14519500732422, -38.75123977661133, 13.619154930114746, 27.440670013427734, 0.2698628604412079, -3.348583221435547, 50.10150909423828, 15.40500545501709, 5.797332286834717, -40.558780670166016, 8.090157508850098, 52.09728240966797, 3.0283777713775635, 7.322903156280518, -30.419723510742188, 46.809112548828125, 18.116024017333984, 41.43235397338867, -37.4019889831543, 58.35382843017578, 21.62001609802246, -46.738990783691406, 49.14519500732422, -2.915419578552246, 6.90926456451416, 49.929012298583984, -30.04471778869629, 9.102072715759277, 37.847206115722656, 15.120203018188477, 83.99214935302734, -75.25965881347656, -47.437042236328125, 2.842189311981201, 30.499948501586914, 32.780181884765625, 39.48614501953125, -18.608497619628906, -10.53453254699707, 16.827463150024414, -20.444976806640625, -5.300902366638184, 21.869197845458984, 6.017582416534424, -7.430356979370117, 46.1876220703125, 31.765844345092773, 44.678646087646484, 38.917022705078125, -13.641533851623535, -23.981651306152344, -8.171846389770508, 9.911290168762207, -48.357547760009766, -76.37364959716797, -12.245564460754395, -7.246690273284912, -39.445030212402344, -4.9553399085998535, -30.606924057006836, 39.80632400512695, 27.05467987060547, -48.822540283203125, 64.44591522216797, -27.11057472229004, 14.722874641418457, -60.73505783081055, 14.308351516723633, 4.227967262268066, -44.84503173828125, -16.49210548400879, 30.44778060913086, -4.0472917556762695, 19.048053741455078, 19.801597595214844, 59.15888214111328, -61.47509002685547, -28.16972541809082, 12.010703086853027, 11.145493507385254, -5.24245023727417, -15.51055908203125, -17.952983856201172, 5.794909477233887, 19.44229507446289, 10.708864212036133, -4.820448875427246, 2.8176918029785156, 31.817670822143555, 23.50790023803711, -32.56100845336914, 17.211374282836914, 3.3822619915008545, 4.636270999908447, 10.841631889343262, -5.959499835968018, -29.960430145263672, -32.102439880371094, 5.542841911315918, 31.93728256225586, -47.6435546875, 23.82064437866211, 46.182621002197266, -23.553985595703125, 24.845556259155273, -31.470836639404297, 47.73463821411133, -14.145956039428711, -17.088611602783203, -19.188514709472656, -1.778845191001892, 80.53942108154297, -65.68119049072266, 9.590863227844238, -0.7483620643615723, -30.572023391723633, -25.771793365478516, 36.08036422729492, 16.57341957092285, -14.682979583740234, -18.41105079650879, 2.9818952083587646, -38.986412048339844, -4.68519401550293, -46.86210250854492, -46.88595199584961, -15.982569694519043, -7.509581089019775, 6.780519485473633, 28.73714256286621, -54.99904251098633, -16.790597915649414, 39.61119079589844, 38.87614059448242, 31.952272415161133, -23.959131240844727, -2.426304340362549, 71.43572998046875, -46.10457992553711, 29.310001373291016, -29.09758186340332, -20.06155014038086, -18.70768165588379, 5.935596942901611, -12.528376579284668, 18.242389678955078, -0.951671838760376, -19.769309997558594, -45.468467712402344, 21.45121192932129, -0.2648048400878906, -36.47585678100586, -62.026573181152344, 27.5838623046875, -6.616214275360107, 7.141485691070557, 6.233057498931885, 12.188302993774414, 44.749027252197266, -35.31575012207031, -14.651591300964355, -15.605857849121094, -22.908605575561523, -19.42011833190918, -2.5560665130615234, 23.758384704589844, 7.338156223297119, -25.043399810791016, -5.236839771270752, -6.42112922668457, -21.42229652404785, 6.607882499694824, -1.8690365552902222, 41.38758087158203, -50.194210052490234, 3.7293522357940674, -38.17445373535156, -12.891865730285645, -6.601049900054932, 5.40875244140625, 5.094366073608398, 17.610225677490234, -26.2164249420166, -46.00517654418945, 0.7410200834274292, -3.060335159301758, 58.85269546508789, 53.942291259765625, -9.700353622436523, 14.958972930908203, -38.34205627441406, -50.77782440185547, -43.97987747192383, -25.996994018554688, -11.859042167663574, 4.946017265319824, -62.6531867980957, 42.23293685913086, 5.495233058929443, -15.97101879119873, 31.804800033569336, 37.61262130737305, -2.632227897644043, 7.335577011108398, 5.206535339355469, 48.98984909057617, -41.67974090576172, 19.73540687561035, -23.76738739013672, 21.33285140991211, 19.34640121459961, 39.01604080200195, 34.54354476928711, -7.454416275024414, 9.53243350982666, 17.193954467773438, 11.74176025390625, 39.745113372802734, 4.949010848999023, -21.723127365112305, -66.677490234375, -33.884090423583984, -53.88560485839844, 17.558053970336914, -34.062374114990234, 38.894065856933594, 24.648691177368164, -16.539011001586914, 1.0137571096420288, -32.224544525146484, -9.276147842407227, 8.987543106079102, -2.472275733947754, -38.45980453491211, 46.72575759887695, 26.642322540283203, 7.727116584777832, -6.393646717071533, -5.798587799072266, -0.5352915525436401, -32.74507141113281, 16.500572204589844, 74.92739868164062, -19.78119659423828, 35.15550231933594, 12.942641258239746, -4.5289411544799805, 21.678691864013672, -12.992692947387695, -52.14482116699219, -3.741339683532715, -11.131858825683594, 0.2560737431049347, 39.97697067260742, -77.51294708251953, -14.335230827331543, -9.14123821258545, 38.02485656738281, -52.766075134277344, 47.93141174316406, -13.663750648498535, -39.74030303955078, -19.596433639526367, 15.884995460510254, -44.87599182128906, 2.0414657592773438, 55.488800048828125, 8.582183837890625, -30.05025291442871, -34.323265075683594, 9.240294456481934, -59.70790100097656, -3.1514551639556885, -43.78638458251953, -29.772274017333984, 1.7955315113067627, 3.5665042400360107, -5.603608131408691, -0.43555817008018494, -17.457246780395508, 30.436452865600586, 24.543670654296875, -49.179412841796875, 34.7862434387207, 7.939905643463135, -5.968639850616455, -0.12774354219436646, 53.24537658691406, -3.776888370513916, -2.06949782371521, -50.0215950012207, 32.29967498779297, -52.976585388183594, 16.21843719482422, -33.32316970825195, -3.6470375061035156, -13.06421947479248, 2.592134952545166, 38.78602600097656, 7.419439315795898, -3.8978347778320312, -0.3445420265197754, -0.6731491088867188, -27.078065872192383, -9.701038360595703, 13.408308029174805, -33.49232864379883, 8.815597534179688, 51.38834762573242, 44.74225997924805, -26.00930404663086, 14.59363079071045, 13.102968215942383, 39.378944396972656, -5.728546619415283, 112.93913269042969, -5.3074541091918945, -0.7850403785705566, -15.025062561035156, 18.334566116333008, 22.367691040039062, 0.520755410194397, 30.144264221191406, 15.32347583770752, -30.147518157958984, 18.15224266052246, -10.911797523498535, -23.088701248168945, 73.59991455078125, 16.31513023376465, 12.042722702026367, 42.67524337768555, -27.019535064697266, 22.223825454711914, 31.908798217773438, -0.709391713142395, 36.69804000854492, 32.37493133544922, 43.644737243652344, -40.74280548095703, 10.162239074707031, 29.783924102783203, -50.875465393066406, 22.615955352783203, 0.17506393790245056, -31.35535430908203, -46.17514419555664, -19.46268081665039, 44.61617660522461, 22.377017974853516, 15.174089431762695, -28.043209075927734, -23.52652359008789, -10.520441055297852, -10.32293701171875, -34.59953308105469, -13.466020584106445, 56.960201263427734, -1.862388253211975, -7.362851142883301, 26.527990341186523, 6.586857318878174, -38.948665618896484, -8.079708099365234, -4.918168067932129, 25.686981201171875, 21.991554260253906, 25.6275634765625, 20.559675216674805, -35.008514404296875, 23.655742645263672, 21.18105125427246, -23.853361129760742, 30.70230484008789, 19.98951530456543, -49.328067779541016, -2.5603346824645996, 10.252044677734375, 58.08796691894531, 13.83432674407959, -38.575992584228516, 13.164790153503418, -24.52753257751465, 25.54932403564453, 48.85179901123047, -12.075641632080078, -9.508214950561523, 6.409029006958008, -12.713933944702148, 15.740225791931152, -12.847485542297363, 53.03141403198242, 21.94496726989746, -19.28738021850586, 10.421769142150879, -21.88509750366211, -39.75597381591797, -21.26213264465332, 19.888076782226562, -38.171730041503906, 15.616241455078125, 44.745723724365234, 3.527409315109253, -40.975013732910156, 8.055492401123047, -48.953250885009766, 5.826698303222656, 12.621240615844727, 31.220474243164062, 48.49199295043945, 19.258211135864258, -27.12258529663086, 7.795234680175781, 41.59416580200195, 27.64688491821289, -16.093523025512695, 23.865943908691406, 20.755041122436523, 42.87351989746094, 23.243064880371094, 17.755695343017578, -31.88275909423828, -55.4875602722168, -16.71440315246582, 3.649664878845215, 3.120727062225342, -3.117655038833618, 50.512229919433594, -39.03495407104492, -6.735213279724121, -14.882112503051758, 97.71276092529297, 11.329773902893066, -9.470606803894043, 9.746408462524414, -15.923219680786133, -17.51710319519043, 24.721811294555664, 20.7814998626709, -58.869380950927734, -62.0795783996582, 42.56336975097656, -21.63037109375, 3.9010918140411377, -13.345577239990234, 32.81396484375, 3.299246311187744, 27.952701568603516, -8.981618881225586, 2.314748525619507, -13.056802749633789, 9.711495399475098, -28.65540313720703, 25.19571304321289, 18.909196853637695, 9.107481002807617, 37.93826675415039, 9.269984245300293, -41.70271301269531, -11.941245079040527, -17.81307029724121, 0.8362808227539062, -16.084758758544922, -8.898816108703613, -21.84355926513672, -52.50985336303711, -1.3227710723876953, -23.187034606933594, -3.0955328941345215, -24.68108558654785, -22.39252471923828, -49.50951385498047, -39.22068405151367, 0.7655515074729919, 10.378832817077637, -14.266307830810547, 18.527393341064453, 4.285348892211914, -37.474365234375, 14.289308547973633, 5.341433048248291, -6.80797815322876, 42.320308685302734, -41.185386657714844, -36.473388671875, 26.265995025634766, -7.419248580932617, 2.8815886974334717, -11.678309440612793, 18.81971549987793, -12.38448715209961, 35.230289459228516, 2.5867419242858887, -22.987655639648438, -11.431884765625, -29.59885025024414, 23.48532485961914, -0.42546606063842773, -49.49654769897461, -55.593360900878906, 20.210805892944336, 7.830806732177734, 3.9785704612731934, 0.007949933409690857, -22.782529830932617, -67.53838348388672, -9.66505241394043, 33.327735900878906, -34.55390548706055, 2.7325453758239746, -42.292381286621094, 16.891311645507812, 16.470252990722656, 11.717511177062988, 3.6026463508605957, -4.637385368347168, 38.32255554199219, -82.12238311767578, -31.895662307739258, -28.89195442199707, -29.46746253967285, 28.70378875732422, 31.004289627075195, -47.88478469848633, -8.121533393859863, -6.9747233390808105, 35.656944274902344, -26.240617752075195, -47.39791488647461, 38.641822814941406, -5.324667453765869, -33.595458984375, 16.65985870361328, -40.49357223510742, -45.74738693237305, 31.757240295410156, 11.74159049987793, 27.580463409423828, 1.179281234741211, 23.995281219482422, 30.83466911315918, -3.968250274658203, -15.146382331848145, 32.971309661865234, -7.129426002502441, 19.39012908935547, 31.643945693969727, 33.07913589477539, -7.3683953285217285, -29.851360321044922, -21.81476593017578, 20.720300674438477, -37.07681655883789, 27.740703582763672, 53.38780975341797, -15.325153350830078, -41.74993133544922, -57.84160614013672, 32.84284973144531, -45.73094177246094, 11.083161354064941, 14.999797821044922, 39.0397834777832, 33.689029693603516, -17.11357879638672, 18.053068161010742, 44.03704833984375, -33.73369598388672, -2.8330888748168945, -23.502464294433594, 1.4707252979278564, 20.939828872680664, -58.231544494628906, -6.586494445800781, 6.26898717880249, -21.11972999572754, 12.660638809204102, -7.433595657348633, -15.515791893005371, 12.236908912658691, -22.5205135345459, 20.47864532470703, -3.709787607192993, -12.688091278076172, -24.268150329589844, 32.44189453125, 7.333751678466797, 15.348301887512207, -34.43840789794922, -6.190216541290283, 24.870920181274414, 1.2620470523834229, 21.791372299194336, -40.42771911621094, 44.31888961791992, -12.636761665344238, -9.481463432312012, 6.543394565582275, 42.32978057861328, -19.811494827270508, 8.971465110778809, -33.80980682373047, 43.18738555908203, -7.749040126800537, -24.901418685913086, -28.660076141357422, 24.12063217163086, -37.64663314819336, 14.482492446899414, 40.94776916503906, -65.68338012695312, -3.5303120613098145, 17.52461814880371, 14.112982749938965, -8.33536148071289, -5.13297700881958, 59.377593994140625, -9.718923568725586, -4.131495952606201, 40.386837005615234, -21.49732780456543, -14.861822128295898, 14.476107597351074, -7.3209452629089355, -33.92695999145508, 19.966089248657227, 17.82486343383789, 22.519418716430664, -18.10144805908203, 6.369503974914551, 24.230815887451172, -30.0126895904541, 2.9251184463500977, 11.108625411987305, 18.636878967285156, -1.1909475326538086, -96.9656982421875, -72.98977661132812, -61.8281135559082, 12.51908016204834, 10.381953239440918, 42.604408264160156, -27.126602172851562, 26.13051986694336, 19.33037757873535, 11.662169456481934, -24.410287857055664, 5.561404705047607, -20.86592674255371, 22.564420700073242, -81.17353820800781, 14.582758903503418, -14.591912269592285, 47.01764678955078, 5.874571323394775, 5.220864295959473, 21.764301300048828, 37.92021942138672, 21.135726928710938, -34.884918212890625, -31.656949996948242, 18.741533279418945, 4.364847183227539, -15.737590789794922, 35.743289947509766, 28.94681739807129, -20.15495491027832, -18.287294387817383, 34.6529655456543, 24.742977142333984, -16.233579635620117, -1.2920866012573242, -2.0038681030273438, 11.56731128692627, -33.94373321533203, 4.499937057495117, -19.06783676147461, 50.90489959716797, -31.284814834594727, 52.96444320678711, -54.64784622192383, 48.89649963378906, -27.199190139770508, -6.894902229309082, -17.251251220703125, 73.13787078857422, -14.891819953918457, 45.537200927734375, 13.617603302001953, 10.750125885009766, -27.415489196777344, 9.078500747680664, 2.3979718685150146, -25.290451049804688, -13.667119026184082, -41.030845642089844, -33.59904098510742, 11.368592262268066, 32.03169250488281, -15.849113464355469, 32.13373947143555, -0.5518180727958679, 44.49781036376953, 36.82748794555664, 34.61066818237305, 3.555065393447876, -34.31760787963867, 21.839475631713867, 22.86835289001465, -13.067726135253906, 37.97713851928711, -52.25232696533203, 12.351821899414062, 10.684911727905273, -49.455875396728516, -40.269309997558594, -28.665178298950195, -28.069618225097656, 47.575767517089844, 22.810922622680664, -5.489366054534912, 24.94544219970703, 11.907573699951172, -39.912899017333984, -49.61848068237305, 6.969572067260742, 25.094640731811523, 2.437075614929199, 48.246639251708984, 35.135162353515625, -17.938295364379883, 34.122989654541016, -13.035367012023926, -19.721954345703125, 22.24908447265625, 11.968911170959473, 29.06386947631836, -1.6610691547393799, -41.17538833618164, 29.337940216064453, 42.24253463745117, -1.6173171997070312, 23.14813995361328, -68.39816284179688, -23.191865921020508, 24.64435386657715, 13.858441352844238, -5.952411651611328, 47.749847412109375, -12.862184524536133, -33.20923614501953, -19.781925201416016, -29.028457641601562, 12.660967826843262, 26.244647979736328, 27.561098098754883, -26.257652282714844, 6.921553611755371, -26.930156707763672, 19.516624450683594, 15.258557319641113, -28.205272674560547, -16.901885986328125, -17.22563934326172, -5.739476680755615, -35.60944366455078, 39.64883041381836, -18.533641815185547, 20.692272186279297, -8.198697090148926, 6.689945697784424, -8.658805847167969, -52.940711975097656, 16.547225952148438, 34.35471725463867, 93.55579376220703, -21.78495216369629, 24.50902557373047, 3.454439401626587, 15.207645416259766, -10.229817390441895, 26.480735778808594, 48.80024719238281, 34.17733383178711, 1.3776493072509766, 1.4490842819213867, 38.91355895996094, -13.886272430419922, -70.09009552001953, -63.66416549682617, -28.972309112548828, 43.92666244506836, 34.944217681884766, 11.945690155029297, -22.58265495300293, 43.79984664916992, -4.944037914276123, -19.013126373291016, -52.4123420715332, 19.397367477416992, -27.021244049072266, -0.6301331520080566, -53.15230941772461, -38.24934387207031, -10.902687072753906, 5.795679569244385, 13.707542419433594, 42.70701599121094, 46.42277908325195, 11.50090217590332, 2.3336644172668457, -33.23596954345703, -27.46885108947754, 11.872049331665039, -11.527393341064453, -22.2156925201416, 20.676427841186523, -17.43451499938965, -34.930755615234375, 4.543107509613037, 35.00685119628906, -32.48018264770508, -7.516873359680176, 16.173904418945312, -24.544525146484375, 11.863245964050293, 65.33296966552734, 23.490175247192383, -60.07852554321289, -39.15945816040039, 46.068119049072266, 85.24629974365234, 23.707788467407227, 10.659477233886719, 41.25559616088867, -12.532553672790527, -17.967897415161133, -32.67272186279297, -47.41300964355469, -26.772642135620117, 16.928691864013672, 2.4864189624786377, 21.798215866088867, 21.87355613708496, 58.52466583251953, 33.54692840576172, 13.536178588867188, 15.250057220458984, 29.489295959472656, -30.716794967651367, -4.070516586303711, 34.93357467651367, -6.6596550941467285, -21.392127990722656, -68.04425811767578, -5.022793769836426, -37.441993713378906, 0.5905535221099854, 30.859546661376953, 2.6180684566497803, 13.750360488891602, -25.391956329345703, 25.25828742980957, -38.61819839477539, 2.412451982498169, 39.28834533691406, -13.682900428771973, 16.266464233398438, -1.1430439949035645, -0.8098580241203308, -15.018342971801758, -11.591626167297363, 48.00873947143555, 32.830875396728516, 0.2570071816444397, -13.19278621673584, 57.5424690246582, -41.88986587524414, -8.04433822631836, -3.042558431625366, -24.26003646850586, 0.6168915033340454, 39.67643737792969, -65.22560119628906, -19.729759216308594, 1.9490556716918945, -31.67291259765625, 4.6552815437316895, 8.033147811889648, -13.563392639160156, -20.138397216796875, -14.34521770477295, 1.2524280548095703, -5.6219258308410645, -17.170042037963867, 25.683347702026367, -56.88677978515625, 70.5321044921875, -45.732818603515625, 43.16514587402344, 44.05030059814453, 5.664608478546143, -31.68987464904785, 65.50651550292969, -38.60341262817383, -23.397327423095703, -15.877013206481934, 18.218280792236328, 70.21051025390625, -2.3528616428375244, 59.60066223144531, 24.34914207458496, 47.2853889465332, 26.253246307373047, 69.04640197753906, -24.39291763305664, -8.091768264770508, 17.525928497314453, 17.233320236206055, 38.50739669799805, -14.752372741699219, 4.224183559417725, -19.555585861206055, -36.793113708496094, 17.090862274169922, -26.13336753845215, -14.016766548156738, -41.58000183105469, -4.080606937408447, -14.202371597290039, 17.748676300048828, 67.26403045654297, -59.3432731628418, -44.91012954711914, -7.336437702178955, -41.30875778198242, -21.222862243652344, 37.48693084716797, -49.81236267089844, -6.836643218994141, -13.720468521118164, 12.862946510314941, -30.232301712036133, 31.779010772705078, -4.072665691375732, 11.427590370178223, 6.874356269836426, 0.7219173312187195, -21.500375747680664, 34.970436096191406, 12.342166900634766, -7.071047782897949, 21.13936996459961, 0.03062957525253296, 21.60660171508789, -43.87709426879883, -1.614919662475586, -9.453590393066406, 28.723217010498047, 13.13888931274414, 29.136974334716797, 18.397171020507812, -35.834442138671875, 2.1200571060180664, -20.506988525390625, -2.9182851314544678, 14.64408016204834, -25.952255249023438, -24.234447479248047, 5.879363536834717, 26.68352508544922, -20.903371810913086, -3.4074959754943848, 28.739421844482422, -38.962642669677734, -25.94120979309082, -37.90263748168945, -27.37246322631836, 42.343162536621094, 66.92279052734375, -22.912944793701172, -13.24438190460205, -15.536600112915039, -38.90620422363281, -47.194766998291016, -11.857885360717773, -34.15338897705078, 23.48362159729004, -33.36033248901367, 20.982425689697266, 22.632274627685547, -36.729068756103516, 33.98765182495117, 6.875119686126709, 36.29952621459961, -8.40970230102539, -17.052501678466797, -22.11941146850586, -23.629261016845703, 16.681854248046875, 69.5530014038086, -4.281365394592285, 40.02732467651367, -24.84126853942871, 18.780467987060547, -30.1363525390625, -25.293882369995117, -74.53357696533203, -0.010741591453552246, 29.125404357910156, -5.081790924072266, -17.595563888549805, -6.527983665466309, -20.23780059814453, -35.919654846191406, -1.3462393283843994, -27.62986946105957, -17.12101173400879, -61.25569152832031, 10.829172134399414, -0.5689413547515869, 19.200868606567383, -5.499777793884277, -41.94618606567383, -36.660770416259766, -21.038066864013672, -8.795385360717773, -4.510682582855225, 26.503726959228516, -10.643338203430176, 7.710600852966309, 30.757001876831055, -29.493608474731445, 1.9944875240325928, 8.506418228149414, -32.93179702758789, 50.09107971191406, 35.683345794677734, -14.65893268585205, 17.85436248779297, -69.10618591308594, 23.270618438720703, 26.120677947998047, 45.83263397216797, -3.898611068725586, -4.92936897277832, 31.17102813720703, -37.53312301635742, 5.6096038818359375, 5.850642681121826, -4.410442352294922, 8.729752540588379, -2.763737678527832, 49.89321517944336, -20.762670516967773, -35.084877014160156, -24.45608139038086, -40.65640640258789, 41.536720275878906, -20.790929794311523, -15.59244441986084, -1.0705652236938477, 4.981353759765625, 58.683162689208984, -20.633689880371094, 1.9217181205749512, 18.22429847717285, 4.755275726318359, 41.206809997558594, 39.86859893798828, -20.726003646850586, -38.2844352722168, -34.1497802734375, -19.360742568969727, -33.56402587890625, 7.3452582359313965, -17.100196838378906, 7.440822601318359, 6.87775993347168, -12.01379680633545, 48.307456970214844, 34.656494140625, 17.6912899017334, 8.540765762329102, -32.080448150634766, 29.55699920654297, -66.59626007080078, 21.068716049194336, -35.93281173706055, -12.2198486328125, -21.280797958374023, 37.6966438293457, 60.00505447387695, 62.42632293701172, -22.707122802734375, 27.872451782226562, -21.658119201660156, 29.219058990478516, 35.170379638671875, 47.2333869934082, -35.723548889160156, 29.148454666137695, -0.005949817597866058, -1.5541671514511108, -8.707600593566895, 9.481401443481445, -32.286434173583984, 0.4881059527397156, -9.870742797851562, 18.3461971282959, -4.17800760269165, 13.682371139526367, -21.923274993896484, -31.76063346862793, -7.6232452392578125, 29.822711944580078, 27.849565505981445, -60.282264709472656, 9.908738136291504, 8.488588333129883, 32.797508239746094, 7.291227340698242, -13.2623872756958, 11.179975509643555, 15.046333312988281, -19.676334381103516, -2.586235761642456, -34.870845794677734, 23.738845825195312, 3.5492136478424072, 22.031211853027344, -44.72956085205078, 20.7882022857666, -1.5436278581619263, 18.551530838012695, -10.556808471679688, 43.26143264770508, 12.672894477844238, -9.30276107788086, -5.081639766693115, 3.6331417560577393, 52.66135787963867, 47.2092170715332, 3.389538288116455, 16.398296356201172, -20.482961654663086, 7.989699363708496, 52.24321746826172, -13.195182800292969, -22.441837310791016, 75.15763092041016, 1.416286826133728, 64.63565826416016, 27.076967239379883, -4.6231913566589355, -58.170101165771484, 37.14754867553711, -11.209461212158203, -45.0046501159668, 3.9781808853149414, 19.462770462036133, -37.56319046020508, -29.11551284790039, 10.969932556152344, 40.63797378540039, 42.10391616821289, 9.345701217651367, 7.694432258605957, 36.16744613647461, 8.866645812988281, -4.554410457611084, 74.96869659423828, 2.2588634490966797, -25.602088928222656, -3.6912918090820312, 1.9306716918945312, 12.624588966369629, 13.370781898498535, -21.68569564819336, -19.931615829467773, -58.38642501831055, -26.83658790588379, -28.237903594970703, -25.481002807617188, 47.920021057128906, -47.26448440551758, 10.200959205627441, -17.454261779785156, -0.5703470706939697, -14.603468894958496, 41.53746032714844, -17.3971004486084, 3.608402967453003, -13.850586891174316, 72.13011169433594, 34.7172966003418, 21.06435775756836, -10.575925827026367, -23.83668327331543, -8.887762069702148, -5.177235126495361, 7.777041912078857, 18.519742965698242, -27.42409896850586, -5.481485843658447, 6.5715227127075195, 12.960806846618652, 23.764039993286133, -22.482280731201172, -60.81570816040039, 12.558762550354004, -0.08299136161804199, 19.38135528564453, -20.90800666809082, -7.260079383850098, -56.79033660888672, -19.859649658203125, -45.55940246582031, -9.290349006652832, -45.61787033081055, 37.75080490112305, 5.081193923950195, 40.184303283691406, 45.997459411621094, -15.151697158813477, 7.284801483154297, 15.331850051879883, -8.454813003540039, -11.305912971496582, -24.36513328552246, -44.00513458251953, -13.620627403259277, -21.603025436401367, 27.469955444335938, 55.5462760925293, 7.966211795806885, -48.751556396484375, -35.46944046020508, -11.388972282409668, 8.625489234924316, 31.364681243896484, 39.30567932128906, 12.631638526916504, -17.050607681274414, 23.344249725341797, -3.667694091796875, -42.66300582885742, 69.69945526123047, -0.6501431465148926, -8.461840629577637, -17.13765525817871, 22.187633514404297, -7.226494789123535, 40.84275817871094, -16.664640426635742, -30.213592529296875, 42.79914093017578, 12.113370895385742, -44.00464630126953, 23.406002044677734, 34.02940368652344, -24.788715362548828, 54.90951919555664, -2.439629554748535, -8.322672843933105, 28.83283042907715, 11.358673095703125, -5.110662937164307, 17.665685653686523, -66.47571563720703, 46.31245040893555, -54.89224624633789, 5.855809211730957, -11.77915096282959, 20.90139389038086, -11.705581665039062, 80.99787902832031, 37.40578842163086, 58.52544403076172, 18.75457000732422, -57.66021728515625, 7.7584638595581055, -10.746469497680664, 20.28754425048828, 20.261428833007812, -3.8770740032196045, -26.860206604003906, 0.4080289602279663, 11.543578147888184, -14.149578094482422, 17.990503311157227, -19.511445999145508, 8.207818984985352, 8.310263633728027, 7.746675491333008, 42.8024787902832, 32.61832046508789, 3.8724253177642822, -11.96196174621582, 36.85783767700195, 11.700799942016602, -27.591909408569336, -21.72475814819336, 33.881561279296875, 36.48121643066406, 8.375138282775879, 6.5865607261657715, 38.847686767578125, 54.094505310058594, -40.97908020019531, 37.0919303894043, -27.045686721801758, -9.28823471069336, 18.618640899658203, -29.286338806152344, 27.755699157714844, 43.989891052246094, 29.162212371826172, -34.85406494140625, -0.8839455842971802, -3.3389039039611816, 39.34363555908203, -12.436535835266113, 90.3347396850586, 14.676698684692383, 3.027068853378296, 4.611718654632568, -0.32905709743499756, -14.982147216796875, 11.851444244384766, -11.474595069885254, 32.14739990234375, -72.01876831054688, -1.130730152130127, 25.45277976989746, 45.730201721191406, 4.813253402709961, -18.627565383911133, -1.9244474172592163, -48.144447326660156, 63.37771987915039, 13.388837814331055, 57.553504943847656, -22.271034240722656, 18.883115768432617, -19.726665496826172, -1.1536340713500977, -27.384756088256836, -4.763738632202148, 49.789920806884766, -0.3807097375392914, -22.25379753112793, 6.018168926239014, 13.327702522277832, 21.261600494384766, 30.662120819091797, 28.50626564025879, 55.704368591308594, -11.039112091064453, 13.742820739746094, -15.939139366149902, -15.261816024780273, -48.47781753540039, 9.155896186828613, 1.1014574766159058, 50.20187759399414, -3.776803731918335, -3.7858145236968994, -13.365458488464355, -42.28433609008789, 49.97077560424805, -7.733771324157715, 3.390577793121338, 15.940914154052734, 0.4123375415802002, 12.438948631286621, 38.144325256347656, 16.141738891601562, 9.964936256408691, -20.68326759338379, -46.25283432006836, -26.66286849975586, -15.833085060119629, -27.37609100341797, -26.35862922668457, 52.60386276245117, -35.807472229003906, -17.145410537719727, 19.802818298339844, -13.846312522888184, 24.540851593017578, -58.24961853027344, 8.080283164978027, -27.092052459716797, 37.60369110107422, -26.578723907470703, -49.3254280090332, 5.12880277633667, -9.872267723083496, 16.686460494995117, -15.866470336914062, 76.95234680175781, 9.132847785949707, -11.905847549438477, 6.709825038909912, 23.162446975708008, 9.94128704071045, 26.66930389404297, 11.353300094604492, -52.55220413208008, 47.58987045288086, 2.3138811588287354, 15.599635124206543, 12.572309494018555, -26.301025390625, 0.2858152985572815, 22.972869873046875, 57.87722396850586, 27.235727310180664, 3.171095609664917, 31.833444595336914, 41.805484771728516, 6.910345077514648, -9.516871452331543, 82.20558166503906, -7.303638458251953, -19.26596450805664, 8.267984390258789, -9.382808685302734, 18.182422637939453, -42.60866928100586, 40.50199890136719, -43.918128967285156, -24.46812629699707, 12.187321662902832, 4.150918483734131, 14.73995304107666, 7.056182384490967, -4.495645523071289, -48.22638702392578, -29.341564178466797, -31.31141471862793, 17.85944366455078, -29.995147705078125, -33.05976104736328, -82.66460418701172, -14.989577293395996, 11.858878135681152, 29.001419067382812, -10.577234268188477, 17.35194969177246, -1.093919277191162, 12.82182788848877, -38.97098159790039, -18.858285903930664, -16.804391860961914, -35.43309783935547, -13.87435531616211, 5.963496685028076, 2.4107165336608887, -2.4667654037475586, 52.312557220458984, -36.77855682373047, 23.55458641052246, 6.0914506912231445, -22.742752075195312, 19.39703941345215, 61.23334884643555, -58.97941589355469, 75.71715545654297, -12.234171867370605, -11.286231994628906, -19.342063903808594, -23.497028350830078, -61.38570022583008, -3.588517665863037, 45.013999938964844, -13.189541816711426, -17.668212890625, -18.951904296875, 11.369366645812988, -46.687225341796875, 20.929424285888672, -19.63216209411621, 28.768314361572266, 2.550752639770508, -56.571022033691406, 51.31208038330078, -9.08479118347168, -3.6369354724884033, 33.04212951660156, 28.998533248901367, 40.67252731323242, 3.0106871128082275, 28.892011642456055, -11.205575942993164, 4.669189929962158, 12.042007446289062, 28.59477996826172, -47.24928283691406, 33.865989685058594, -35.71508026123047, -11.207473754882812, -8.841442108154297, 49.03486633300781, 54.0059928894043, 22.7874755859375, 5.953512668609619, 14.639959335327148, -26.388917922973633, -37.05088806152344, -24.05687141418457, 54.230926513671875, 7.917159557342529, 1.71400785446167, -0.25342321395874023, 35.52128219604492, 25.089630126953125, 9.833404541015625, -46.28045654296875, 12.794092178344727, 5.167083263397217, -18.32282257080078, 43.47663497924805, -36.5163459777832, 19.397008895874023, -33.724815368652344, 9.696965217590332, -10.540087699890137, -52.53156280517578, 10.33371353149414, 10.857728004455566, 35.039039611816406, 9.723062515258789, 24.185434341430664, 21.493173599243164, -13.762084007263184, 44.174835205078125, 33.2335205078125, 43.11091995239258, 6.846500396728516, 52.7686882019043, -60.08924865722656, -23.28881072998047, -8.329190254211426, 57.180965423583984, 25.38274574279785, 47.25782775878906, -26.685976028442383, 1.9140328168869019, -39.0612907409668, -2.849806308746338, -18.262588500976562, 39.2452507019043, 7.002572536468506, 3.338756561279297, 62.90504455566406, -21.371057510375977, -31.07278060913086, -9.687880516052246, 3.011242389678955, 35.8311653137207, 84.28546905517578, -2.903055191040039, -59.11162185668945, 31.377826690673828, -25.40437889099121, 33.051780700683594, -18.70662498474121, -27.73712921142578, 19.228961944580078, -33.9416389465332, -16.27888298034668, -16.529571533203125], [-22.931636810302734, 18.97802734375, 9.550909042358398, 38.4904670715332, 61.940853118896484, -7.875969409942627, -30.748361587524414, 4.152828216552734, -22.221824645996094, 0.2836129665374756, 0.03165347874164581, 34.47855758666992, 11.156952857971191, -43.74506378173828, -9.079658508300781, 2.8639731407165527, 27.5037899017334, -65.9646987915039, -27.561851501464844, -21.715940475463867, 3.70902681350708, -8.49535846710205, -10.632267951965332, 11.501028060913086, 67.0330581665039, -22.899410247802734, -88.04845428466797, 16.840370178222656, -17.056236267089844, -45.509788513183594, -12.549625396728516, 10.204257011413574, -13.644214630126953, -11.448684692382812, -15.606307029724121, -24.662912368774414, 92.66714477539062, 22.350337982177734, 39.05165481567383, 47.499290466308594, 12.979337692260742, 13.676265716552734, -55.37792205810547, 7.98008394241333, 5.156681537628174, 45.67575454711914, -74.1181411743164, 16.662506103515625, -10.97714900970459, -49.87055206298828, -1.6437878608703613, 24.459550857543945, -19.23823356628418, -16.675830841064453, -27.7377986907959, 46.98624038696289, 7.495771408081055, -14.11806869506836, 19.650394439697266, 52.90622329711914, 30.68327522277832, -10.955771446228027, 0.5506356954574585, 43.40171813964844, 2.208559989929199, 30.40963363647461, -18.96895408630371, 11.805767059326172, 5.322590351104736, 12.095882415771484, 27.49635124206543, -26.897815704345703, 70.38052368164062, 14.698288917541504, -45.088497161865234, -7.942078113555908, -21.115209579467773, -10.570923805236816, 7.11377477645874, 4.212193489074707, 6.066804885864258, 20.476215362548828, -52.72550964355469, 32.826725006103516, -17.111173629760742, 15.130096435546875, 5.443907737731934, 27.988670349121094, -28.09742546081543, -23.793991088867188, -0.8269737958908081, -5.1433634757995605, -46.28548049926758, -47.956260681152344, -17.27422332763672, 18.03194808959961, 9.357335090637207, 12.588325500488281, 25.61237335205078, -22.41908073425293, 21.793426513671875, 48.15830612182617, -16.631738662719727, -11.22786808013916, 1.6594043970108032, -5.30081033706665, -59.92630386352539, -3.878235340118408, -34.438419342041016, 18.370555877685547, -2.3793368339538574, 7.331533908843994, -19.060964584350586, -2.116197109222412, -34.460853576660156, -19.625791549682617, 1.6504709720611572, 49.36738967895508, 5.0446295738220215, -38.047698974609375, 7.696259498596191, 4.417394638061523, -25.004056930541992, -0.22389939427375793, -10.028822898864746, 43.475921630859375, 5.2735676765441895, 42.932037353515625, -16.3088436126709, -12.039628028869629, -13.479931831359863, 42.672489166259766, 28.122983932495117, 13.005095481872559, 11.57889461517334, -26.06951904296875, -11.965560913085938, -7.41412878036499, -8.817961692810059, -3.765352725982666, 40.98066711425781, 2.3772032260894775, 27.24749755859375, -29.134136199951172, -18.29930877685547, -5.824005603790283, 57.828338623046875, 31.027393341064453, 25.920093536376953, -29.10515022277832, 13.449468612670898, -22.968822479248047, 17.57870101928711, 28.194469451904297, 58.56303024291992, -6.704612731933594, 23.3095760345459, -18.682832717895508, 11.434165000915527, -21.018831253051758, -0.9750570058822632, -10.165124893188477, -0.7492709159851074, -27.77132797241211, -28.793977737426758, -27.444578170776367, -32.25541687011719, 37.242210388183594, 51.191864013671875, -68.16336059570312, -12.52542495727539, -26.337387084960938, 20.090070724487305, -38.66267776489258, 64.97404479980469, 37.47803497314453, -35.05763626098633, -40.892940521240234, 8.503768920898438, 15.296194076538086, -56.758331298828125, -3.7782042026519775, -5.298602104187012, -26.489013671875, 0.15885160863399506, -25.393760681152344, 1.8185824155807495, 0.3987983465194702, 2.5858333110809326, 5.902322769165039, -29.746105194091797, 7.061237335205078, 26.002321243286133, -1.5790677070617676, -1.4794626235961914, 8.984832763671875, -19.023923873901367, 23.262601852416992, -52.872737884521484, 2.281541585922241, 17.599925994873047, 0.8062731027603149, -31.971805572509766, -45.67710494995117, 0.16519081592559814, -54.829139709472656, -1.20320725440979, -7.344542980194092, -7.6263628005981445, -16.189922332763672, 15.34851360321045, 5.682519912719727, -0.8749158382415771, 38.568504333496094, -10.328057289123535, -46.0688362121582, -7.599973201751709, -12.627979278564453, -10.359857559204102, -14.4419527053833, -8.558219909667969, -6.878679275512695, 15.3189697265625, -21.39202308654785, -7.3756022453308105, 25.21282386779785, -25.664064407348633, 7.608986854553223, -1.422658085823059, 14.048476219177246, -20.11948585510254, 34.03361511230469, 19.498231887817383, -74.57689666748047, -28.62603187561035, 13.138432502746582, -8.993948936462402, -25.830984115600586, -34.351531982421875, 34.4515495300293, -38.56584548950195, 16.581079483032227, -17.45012855529785, -40.28471374511719, 11.011526107788086, -12.7752685546875, 23.65150260925293, -38.43994140625, -35.85904312133789, -22.855003356933594, 29.082744598388672, -28.101884841918945, 75.9738998413086, 23.578359603881836, -16.082571029663086, -43.23302459716797, 28.913543701171875, 19.9145565032959, 36.98501205444336, -6.298428058624268, 20.032604217529297, -15.040916442871094, -4.843612194061279, -18.506622314453125, -85.26285552978516, 16.725605010986328, 36.30783462524414, 48.972835540771484, 19.264812469482422, 2.0794646739959717, -35.65818786621094, 59.12400436401367, -9.194772720336914, 10.837638854980469, 14.91823673248291, -29.2311954498291, -21.59123420715332, 17.60456657409668, 26.50714874267578, 26.668132781982422, 4.889848709106445, -24.21031951904297, 72.1014404296875, 27.65726661682129, 15.513602256774902, -2.6218008995056152, -14.083208084106445, 9.371191024780273, -36.60918045043945, 39.478729248046875, -38.29884338378906, 36.77823257446289, -11.928566932678223, 31.511241912841797, 34.12071990966797, 14.65797233581543, 59.96406936645508, 48.702842712402344, 19.147336959838867, -25.659212112426758, 4.302090167999268, 15.521727561950684, 32.16421127319336, -5.924280643463135, 31.643285751342773, 22.554332733154297, 30.472421646118164, 10.813626289367676, 55.59840393066406, 2.1849350929260254, -31.982803344726562, -3.5110907554626465, 58.84907150268555, -46.78608322143555, -55.77354049682617, -5.315732955932617, -16.884624481201172, -25.241640090942383, 24.54639434814453, 7.577299118041992, 12.897687911987305, 6.903624057769775, -5.179478168487549, -0.500148594379425, 16.055927276611328, -45.958560943603516, 11.138278007507324, 44.599021911621094, -31.016521453857422, 13.451254844665527, -7.499392509460449, 35.14256286621094, -17.487037658691406, -16.677194595336914, 6.180583477020264, -13.151374816894531, 1.6331350803375244, 89.07237243652344, -2.472784996032715, 26.556325912475586, -41.67516326904297, -14.580201148986816, 26.93443489074707, -4.977660179138184, 6.991538047790527, 26.349170684814453, -4.71622896194458, 32.56653594970703, 3.42525577545166, 23.76042366027832, 22.817062377929688, 5.863193988800049, -24.930370330810547, 7.399865627288818, -23.049421310424805, 32.64299011230469, 9.052038192749023, -62.7210693359375, 33.36803436279297, -51.964202880859375, -1.5313239097595215, 45.84505844116211, -13.291828155517578, -39.47747802734375, 21.039939880371094, -17.98166275024414, -8.588475227355957, 7.481443405151367, -3.5283215045928955, 0.8604187965393066, -34.984825134277344, 2.033255100250244, 3.9011693000793457, -14.709617614746094, 0.9570956826210022, -22.413930892944336, 9.322572708129883, 16.476058959960938, 27.00333595275879, -19.892009735107422, -4.284966945648193, 10.664031982421875, -30.159086227416992, -10.94519329071045, 39.23319625854492, 31.27729606628418, -10.605828285217285, 12.103928565979004, 26.59954261779785, -12.086541175842285, 15.746366500854492, 52.57527542114258, 23.38422966003418, -26.78713607788086, -18.753808975219727, 1.8419318199157715, 6.941812515258789, -13.120203971862793, 13.522571563720703, -1.7797571420669556, 14.22213363647461, -47.574161529541016, -31.97514533996582, 21.96129608154297, 3.283710479736328, 45.63911437988281, -34.26838302612305, -20.34183120727539, 39.24741744995117, 72.84264373779297, -21.66054344177246, 28.38944435119629, 41.57976150512695, -7.038379669189453, 52.19164276123047, 18.233043670654297, -31.85107421875, -42.41156768798828, -3.908646583557129, 16.901721954345703, -25.465669631958008, 38.326473236083984, 28.20329475402832, 11.26181411743164, -5.626192569732666, 0.8793513774871826, 15.729416847229004, 38.77723693847656, 4.199974060058594, 14.98156452178955, 15.77138614654541, -53.2104377746582, 14.16333293914795, 41.98857498168945, 16.32588005065918, -10.966598510742188, 24.24241828918457, 8.588457107543945, -5.616006851196289, -16.18669891357422, -30.89844512939453, 14.909333229064941, 28.77027130126953, 10.079998016357422, -12.331189155578613, -9.328618049621582, -54.88117599487305, -57.52021408081055, 19.740196228027344, 15.590446472167969, -33.43633270263672, 31.633203506469727, 25.78199005126953, -15.544909477233887, -1.4174071550369263, 20.830425262451172, -37.68350601196289, -20.34454917907715, 21.11589241027832, -10.98918628692627, -5.972825050354004, -14.144705772399902, -25.008668899536133, -14.363719940185547, -30.51198959350586, -83.36228942871094, -32.25371170043945, 5.720919132232666, 11.6212739944458, 63.67301559448242, -16.357494354248047, -28.829851150512695, 1.7086750268936157, -21.899614334106445, 40.177162170410156, 28.67000961303711, -17.934221267700195, -38.61449432373047, -11.942320823669434, 36.34960174560547, -0.2787885069847107, 14.123882293701172, -36.1844367980957, -38.914451599121094, 25.944250106811523, 20.779478073120117, -21.62181282043457, -45.005210876464844, 31.249347686767578, -19.42217445373535, -4.327181339263916, 24.79988670349121, 26.600980758666992, 62.29470443725586, 5.426954746246338, -23.720529556274414, -2.256782054901123, 42.25846481323242, 39.36614227294922, 41.15264892578125, 17.677291870117188, 46.038108825683594, -53.51078414916992, 16.258010864257812, -34.34644317626953, 15.97358512878418, 38.59327697753906, 35.765403747558594, 5.068733215332031, 13.313680648803711, -17.490102767944336, -13.613658905029297, 2.8729701042175293, -26.625560760498047, 49.25832748413086, 11.146721839904785, -16.459270477294922, -8.394822120666504, 42.68022155761719, 16.177520751953125, -28.451740264892578, 10.397239685058594, 19.23671531677246, -12.101712226867676, -14.64156723022461, -24.15838623046875, 35.60003662109375, -13.524309158325195, 14.930039405822754, 38.312599182128906, -22.98992156982422, -5.771158695220947, 27.847368240356445, 12.534149169921875, 87.09395599365234, -12.851293563842773, -32.94797897338867, 5.85967493057251, -1.659501314163208, 33.298866271972656, 9.620003700256348, 32.10258483886719, -1.1426818370819092, 60.237144470214844, -0.8182473182678223, 25.036012649536133, -29.839412689208984, -17.912023544311523, 1.4109710454940796, 21.48000144958496, -51.33495330810547, -47.4582405090332, 24.506994247436523, -3.5222978591918945, -28.75200080871582, 13.273811340332031, -2.7880311012268066, -36.95857238769531, 14.589316368103027, -16.18128776550293, -45.71613311767578, 0.38410866260528564, 26.137924194335938, 33.93244934082031, -7.789393901824951, -35.60398864746094, 62.2186164855957, -12.991896629333496, 2.7074174880981445, 44.23499298095703, -37.59239196777344, -8.339359283447266, 8.284592628479004, 0.5795332789421082, -35.15174865722656, -6.603244304656982, 52.73760223388672, -1.4988011121749878, -51.87849044799805, -42.165565490722656, -3.3529672622680664, 12.209211349487305, 5.07805871963501, -29.924903869628906, 52.8647346496582, 54.742958068847656, -19.219755172729492, 17.41417121887207, 42.10162353515625, -5.192054271697998, 17.41729164123535, 13.486958503723145, 35.705474853515625, -11.445782661437988, 17.70355224609375, 41.584449768066406, -21.20068359375, -14.297598838806152, 22.931840896606445, 10.960273742675781, 15.835086822509766, -49.73160171508789, -37.09112548828125, 23.20563507080078, -26.85997200012207, -46.158653259277344, -61.685691833496094, 20.445659637451172, 18.376556396484375, 70.31507873535156, 23.99789047241211, 13.6947021484375, 7.321862697601318, -26.488815307617188, -11.133092880249023, 28.1171932220459, -5.153345584869385, 8.055984497070312, 10.501036643981934, -32.988765716552734, 9.214179992675781, 20.724241256713867, 0.6571859121322632, -4.659948825836182, 18.984336853027344, -68.81303405761719, 28.039363861083984, 25.646085739135742, 41.01429748535156, 37.70065689086914, -59.69684600830078, 16.43449592590332, 20.227493286132812, -58.23579406738281, -49.639583587646484, 29.050460815429688, 39.82661819458008, -3.2817132472991943, 28.1274471282959, -25.422475814819336, 10.33958911895752, -48.74421691894531, 13.975951194763184, -0.8170796632766724, 32.66789627075195, -25.868011474609375, -0.35355931520462036, -33.82918167114258, 0.8685844540596008, -24.67335319519043, -48.41465759277344, 2.5299761295318604, 20.37358856201172, -16.33894920349121, -41.74752426147461, -16.889205932617188, -40.46851348876953, -15.284035682678223, 7.752572536468506, 29.644981384277344, 5.1656575202941895, -17.2185001373291, 3.4761266708374023, -56.954376220703125, -7.9529852867126465, 19.934165954589844, -8.002090454101562, 3.7227399349212646, -46.820743560791016, -58.8167610168457, -4.236095428466797, 40.45397186279297, -24.6463565826416, -17.99156951904297, -47.331016540527344, 39.16667175292969, 11.154474258422852, -5.248263835906982, -73.63297271728516, 5.54906702041626, 13.21694278717041, 25.298307418823242, -9.382792472839355, 52.54511642456055, 47.03075408935547, -6.429202079772949, 11.4699125289917, 33.688899993896484, -36.993038177490234, 60.373985290527344, -16.246732711791992, -13.425097465515137, -8.681432723999023, 11.701278686523438, 13.465594291687012, 22.586164474487305, -57.570228576660156, -23.90233039855957, -66.83479309082031, -90.55001831054688, 34.66061019897461, -16.378358840942383, -27.772668838500977, -12.483978271484375, -26.998306274414062, -29.1375732421875, 24.632164001464844, 11.490028381347656, 41.747047424316406, 1.1068611145019531, -8.451711654663086, -2.9865336418151855, -37.40198516845703, -22.275793075561523, 33.22425079345703, -4.221027374267578, -42.743465423583984, -63.95441818237305, 41.750091552734375, 27.02198028564453, -51.120513916015625, -9.869684219360352, 26.516700744628906, 32.14179992675781, 14.677412033081055, -12.613922119140625, -37.6839714050293, 63.80323028564453, 60.710262298583984, 42.30205154418945, -19.73869514465332, 19.352554321289062, -22.10564613342285, 2.574471950531006, 7.630978584289551, 50.27813720703125, -25.19021987915039, -23.003751754760742, -15.126399993896484, 19.043926239013672, -37.147972106933594, 0.37925976514816284, -32.05794906616211, 18.444557189941406, -5.966790199279785, 25.96986961364746, 31.66245460510254, 10.967606544494629, 52.69490432739258, 26.873973846435547, 76.93733215332031, 36.067054748535156, -16.432172775268555, -26.941343307495117, 26.865219116210938, 15.057265281677246, -8.133403778076172, -12.67448902130127, 13.384712219238281, 15.647363662719727, -34.7682991027832, 0.31839442253112793, -0.3953668475151062, -20.667835235595703, -38.518272399902344, 47.04679489135742, 36.78605651855469, -38.81558609008789, -13.715782165527344, -12.852394104003906, -20.71663475036621, 24.99307632446289, -29.051841735839844, -19.701059341430664, -13.608848571777344, 2.064880847930908, 30.476909637451172, 6.064977645874023, 17.552236557006836, -60.54737854003906, -19.96504783630371, 8.552912712097168, -0.3274243175983429, -10.68179702758789, -16.519176483154297, -26.497615814208984, -47.89432907104492, -19.617521286010742, 34.14488983154297, 4.715330600738525, -38.09905242919922, -11.85951042175293, 5.903744697570801, 52.12593460083008, -54.545291900634766, -0.1765652894973755, 5.778439044952393, 3.248490333557129, 47.7745361328125, 5.847611427307129, 48.002464294433594, -4.172394275665283, 41.00045394897461, 14.40145206451416, -17.548171997070312, -4.099896430969238, 16.533546447753906, 26.820438385009766, 10.693077087402344, 14.697518348693848, 40.997528076171875, -12.476913452148438, -52.528812408447266, -4.62672758102417, 28.893766403198242, -3.7167935371398926, 9.63679313659668, -23.207134246826172, -10.686190605163574, -2.1027252674102783, 50.43251037597656, 4.594143867492676, 43.15473937988281, -10.354228973388672, -20.270034790039062, 4.789870262145996, 11.946599006652832, 54.56110763549805, -68.15027618408203, 11.554804801940918, -35.74689483642578, -62.04340362548828, 7.411073207855225, -10.288302421569824, -8.848533630371094, -12.416871070861816, -40.88268280029297, 40.05137634277344, 28.153295516967773, 30.23017120361328, 1.9330226182937622, -3.7971231937408447, 7.834945201873779, 10.743175506591797, 7.518442153930664, -24.104143142700195, 17.749610900878906, 40.30411148071289, -0.47796526551246643, 22.952640533447266, 38.471282958984375, -13.52005386352539, -8.426726341247559, 15.990727424621582, -34.65959548950195, 40.1728515625, -13.860868453979492, 11.903714179992676, -0.04878652095794678, 6.071876525878906, -3.9881677627563477, -10.544794082641602, 25.60637092590332, 4.694280624389648, 16.88515853881836, -17.310882568359375, 34.84035110473633, -27.11780548095703, 15.379228591918945, -33.63174819946289, -2.454115152359009, -42.9819221496582, -8.179118156433105, -10.729815483093262, -39.12752151489258, 15.956341743469238, -2.7843410968780518, 28.731279373168945, 0.2022116780281067, -9.557701110839844, 27.683935165405273, -52.77125549316406, -21.088991165161133, 34.80622863769531, 13.202628135681152, 15.02902603149414, 9.211134910583496, -1.6193927526474, -65.9601058959961, 13.09850025177002, -13.726011276245117, -22.65784454345703, -28.119029998779297, -25.123756408691406, -14.916014671325684, -32.188480377197266, -41.06106185913086, 38.875125885009766, -36.134735107421875, -24.793663024902344, 8.5907621383667, 10.169434547424316, -22.06440544128418, -0.9186511039733887, -11.280448913574219, -7.3980231285095215, -32.34239959716797, 1.1061601638793945, -1.6490745544433594, 12.974628448486328, -29.994529724121094, 41.532073974609375, -27.14105796813965, 9.972639083862305, 42.49763488769531, 31.155437469482422, 6.372896194458008, 9.183484077453613, -68.70581817626953, -34.491580963134766, -5.258984088897705, 2.8897616863250732, -49.47726058959961, 17.295637130737305, 11.719833374023438, -22.67786407470703, -0.7095266580581665, 49.20954513549805, 22.911224365234375, 6.825523376464844, -7.693502426147461, -30.985774993896484, 16.358957290649414, -30.64065933227539, 18.699108123779297, -15.522775650024414, 9.993020057678223, -58.406185150146484, -3.7703440189361572, 32.74739074707031, 20.207107543945312, -13.637224197387695, -18.511703491210938, -7.6997246742248535, 6.795377731323242, -8.556224822998047, -0.8540352582931519, -14.570490837097168, -28.330482482910156, 55.284820556640625, -11.343605995178223, 3.4961025714874268, 49.5820426940918, -11.414270401000977, -4.017363548278809, 2.7356135845184326, -4.611526966094971, -36.33192443847656, 37.43639373779297, -12.5228853225708, 4.511171817779541, 35.132728576660156, 30.018251419067383, 20.843568801879883, -55.94109344482422, 18.570497512817383, 51.881919860839844, 1.4046313762664795, 12.103671073913574, 8.429183959960938, -38.64524841308594, -48.33627700805664, -47.69953155517578, 4.385138511657715, -37.21721649169922, -14.970359802246094, -13.583073616027832, 42.04848861694336, 6.55866003036499, -16.173044204711914, -14.767032623291016, 23.34347915649414, -31.06619644165039, -57.69255065917969, -25.217205047607422, -9.808659553527832, 24.950450897216797, -5.122582912445068, 2.8381307125091553, -19.597864151000977, -14.615901947021484, 52.95805358886719, -53.92436599731445, 5.582342147827148, 3.573859214782715, 60.60539627075195, -22.565366744995117, -42.8365478515625, -2.5652740001678467, 33.90576934814453, 20.400922775268555, -39.898162841796875, -17.47539520263672, -14.857850074768066, 17.869964599609375, 49.14360809326172, 3.33069109916687, -39.0843505859375, 59.04945755004883, -7.968775749206543, -41.14849090576172, -50.580509185791016, -24.272903442382812, 28.136886596679688, -11.253035545349121, 50.04970932006836, -11.167510032653809, 0.3246508836746216, -29.075008392333984, -1.3993682861328125, -12.162476539611816, -2.7476656436920166, -12.171199798583984, 32.68251037597656, 36.220909118652344, -25.66901397705078, -49.87995529174805, -2.776846408843994, -19.56357765197754, -0.8276386260986328, 41.6727409362793, -9.431571006774902, 11.6737060546875, 19.736265182495117, 47.88187789916992, 20.83441925048828, -24.913114547729492, -29.603052139282227, -27.76727867126465, 4.277162551879883, -13.508612632751465, -13.285164833068848, -6.3108062744140625, 1.6367121934890747, -13.195756912231445, -41.474647521972656, 7.132230281829834, 33.671836853027344, 38.4706916809082, 20.66372299194336, 70.94058227539062, -65.88397979736328, -24.632240295410156, 31.67545509338379, 16.526670455932617, 23.843538284301758, 7.856197357177734, 74.78338623046875, 45.828712463378906, 8.916401863098145, -64.40599822998047, -43.9964485168457, 29.424226760864258, 78.59364318847656, -8.273011207580566, 18.58241081237793, -17.11376190185547, -21.849382400512695, 16.33922576904297, 1.000560998916626, -42.113929748535156, -67.23019409179688, -3.8788444995880127, 80.98680114746094, 14.654086112976074, -14.045378684997559, 65.17140197753906, 35.59336853027344, 20.203582763671875, 2.947650671005249, -15.561423301696777, -21.558807373046875, 1.582999348640442, -9.483549118041992, 5.086971282958984, -26.267826080322266, -34.88096237182617, 9.817525863647461, 9.279520988464355, 37.57956314086914, -7.024798393249512, -24.99627685546875, 21.580801010131836, -17.11738395690918, 12.4963960647583, 36.7900276184082, -18.041484832763672, -11.853684425354004, -11.495368957519531, -11.16524600982666, 5.735078811645508, 14.109238624572754, -3.1772987842559814, 8.192883491516113, -5.820478916168213, -50.872562408447266, -10.508461952209473, 26.052978515625, -25.67058563232422, -22.94293785095215, -2.1570346355438232, 13.405084609985352, 9.961970329284668, 22.53290367126465, 50.001708984375, 10.47326374053955, -10.81246280670166, 2.2234723567962646, 41.484840393066406, -5.040176868438721, -16.09778594970703, 24.362201690673828, 0.41377824544906616, 23.39451026916504, 14.626770973205566, -22.130905151367188, -12.336483001708984, 23.749650955200195, -21.554006576538086, -34.32063293457031, -79.57479858398438, 1.6335057020187378, -14.030843734741211, -0.8500701189041138, 5.699416637420654, -51.680904388427734, 26.918231964111328, -42.312259674072266, 13.430937767028809, 0.6930575370788574, -7.240738391876221, -27.31100082397461, 38.371524810791016, 8.623348236083984, -26.23038101196289, 15.283658027648926, 9.172799110412598, -10.4515962600708, 6.274810314178467, -13.367046356201172, -19.031633377075195, 13.636613845825195, 3.293654441833496, -6.31589412689209, -27.628177642822266, -20.441740036010742, 16.13662338256836, -56.686710357666016, -16.92724609375, -36.77009582519531, -0.7882792353630066, -55.61104202270508, 33.372764587402344, -18.64529037475586, 21.35552215576172, 24.46599578857422, 1.176991581916809, 8.073654174804688, 32.390235900878906, 18.865217208862305, -9.292587280273438, -8.419646263122559, -24.478675842285156, -18.32603645324707, 21.39744758605957, -1.5090956687927246, -12.487689018249512, -53.4000358581543, 39.69910430908203, 6.531970024108887, -37.12946701049805, -56.12266159057617, -8.360467910766602, 22.228967666625977, 29.644718170166016, -29.143335342407227, 24.475326538085938, -48.71849060058594, 17.862138748168945, -0.05709123611450195, 2.9630563259124756, -50.258453369140625, -40.75947952270508, -14.458389282226562, -4.650459289550781, -13.341700553894043, 14.438416481018066, 25.104188919067383, 16.75723648071289, 76.35232543945312, -16.1943416595459, -46.45488739013672, 10.364437103271484, -9.558013916015625, 41.939579010009766, -10.0675687789917, 2.6417384147644043, 2.944542646408081, -9.284666061401367, 8.92844009399414, 64.5748062133789, 35.502159118652344, -68.50579833984375, 10.837376594543457, -19.751285552978516, -67.16345977783203, 39.39151382446289, -22.011600494384766, -7.130623817443848, -67.1423110961914, -29.217041015625, -78.29850769042969, -4.193338871002197, -16.614356994628906, 43.526702880859375, 24.04855728149414, 4.492376804351807, -43.2994499206543, -8.047667503356934, 4.629006385803223, -9.22053337097168, 22.147192001342773, -21.539913177490234, 14.370535850524902, 7.855908393859863, 72.80142974853516, 6.248849868774414, 16.2453670501709, -47.55146026611328, -26.00286865234375, 6.735483169555664, -17.50824737548828, -23.17656135559082, 8.290133476257324, -40.704586029052734, -7.000992298126221, 24.027603149414062, -4.215791702270508, -41.389713287353516, 20.29678726196289, -15.180526733398438, -11.153422355651855, -21.58331871032715, 3.365309715270996, 3.881678819656372, 5.629512310028076, 33.6926155090332, 23.3646297454834, -7.334875583648682, 1.6071494817733765, 1.246609091758728, -21.647432327270508, -17.81918716430664, 13.900293350219727, -35.41051483154297, 36.14596176147461, 66.90068817138672, -35.05482482910156, -9.617775917053223, -8.536827087402344, 5.193052291870117, -33.77293014526367, -24.13370132446289, 1.421271562576294, 3.4399304389953613, -26.724395751953125, 35.806400299072266, 21.532163619995117, 43.593441009521484, -55.47444152832031, -40.49458694458008, 62.73124313354492, 0.40540552139282227, 41.69377517700195, 28.738672256469727, -32.2856330871582, 7.879771709442139, 14.89588737487793, -9.077012062072754, 3.4847304821014404, -2.6358933448791504, 18.739219665527344, 41.134788513183594, -38.447784423828125, 0.2418440580368042, -45.66184616088867, -30.434072494506836, 5.369953155517578, -54.99563980102539, 9.50780200958252, -38.13211441040039, 42.572471618652344, 6.187839031219482, 10.338240623474121, 16.00808334350586, 13.27790641784668, 15.805526733398438, 12.73857307434082, -49.837806701660156, 15.3173246383667, 43.565670013427734, -34.52808380126953, 1.8001086711883545, -44.02385711669922, -17.28669548034668, -15.88245964050293, -4.913808345794678, -18.221704483032227, 23.770858764648438, -23.25470542907715, -22.042675018310547, -9.9201078414917, 26.006132125854492, 33.57126998901367, -35.695159912109375, 15.84334659576416, 1.2582448720932007, 24.245853424072266, -23.75447654724121, 7.1753668785095215, -3.4676215648651123, -6.22692346572876, 23.44538688659668, 25.022005081176758, -13.282089233398438, -8.019856452941895, 13.247386932373047, -20.903282165527344, -52.348358154296875, -18.046775817871094, -10.790165901184082, -14.521785736083984, -44.51458740234375, 5.285940170288086, -8.163863182067871, -8.725147247314453, -32.23606872558594, 42.249229431152344, 48.06996154785156, 7.028306007385254, 7.309647083282471, -7.60679817199707, -6.51667594909668, 20.036685943603516, -16.665651321411133, 16.407726287841797, 29.035924911499023, -12.8702392578125, 55.076141357421875, -0.532232403755188, 57.24059295654297, 31.34160614013672, -20.603757858276367, 39.79800033569336, 12.07326889038086, 45.498626708984375, 9.244029998779297, 41.955596923828125, -59.96947479248047, -13.829880714416504, -21.264310836791992, 12.375794410705566, -27.859874725341797, -19.765653610229492, 4.520065784454346, 16.3018798828125, -45.413963317871094, 31.65694236755371, -48.9827766418457, 1.3705615997314453, 26.925880432128906, -28.823022842407227, -1.2114183902740479, -18.488462448120117, -7.256101131439209, -7.655191898345947, 5.987459182739258, -1.1092720031738281, 19.187822341918945, 59.83101272583008, 8.716217994689941, 17.840221405029297, -1.7793647050857544, -11.137779235839844, -74.11190795898438, -18.0426082611084, 13.82707691192627, -17.948402404785156, -11.91244125366211, -1.7944318056106567, 2.827836275100708, -8.002408027648926, -63.788841247558594, -26.68625831604004, 7.550302505493164, 3.191577434539795, -33.0700569152832, 26.007417678833008, 12.318687438964844, -18.02606773376465, 30.068973541259766, 33.59043884277344, -20.637266159057617, 0.791923999786377, 36.164268493652344, 17.797687530517578, 32.56977462768555, -68.52494049072266, -26.302629470825195, 30.778305053710938, 11.530255317687988, 4.272824287414551, -14.79910659790039, -6.200974464416504, -31.210805892944336, -54.37134552001953, -18.686416625976562, 4.30298376083374, 16.324743270874023, -1.0435919761657715, 11.195465087890625, -12.254603385925293, 19.86165428161621, 47.225074768066406, 33.81700134277344, 40.97099304199219, 17.3719539642334, -54.32005310058594, -19.071882247924805, 38.2579231262207, 36.308380126953125, 11.980093955993652, 37.907920837402344, -50.541526794433594, 48.47580337524414, 34.40669631958008, -58.002769470214844, -36.1005859375, -39.011653900146484, 37.82606506347656, -24.80832290649414, 11.42618465423584, -55.30550765991211, 19.57269287109375, 27.633501052856445, 10.231063842773438, 9.462586402893066, -27.088855743408203, 11.618560791015625, -4.498901844024658, -12.040799140930176, -6.170855522155762, 30.303573608398438, 11.681100845336914, -21.077320098876953, 36.354347229003906, -80.11289978027344, 2.658982276916504, -28.82642364501953, 63.7357292175293, 11.474898338317871, -0.9497170448303223, -21.176010131835938, 52.097110748291016, 40.241477966308594, 31.695158004760742, -9.722966194152832, 8.736833572387695, -4.575185298919678, -30.374713897705078, -34.095130920410156, 2.245060443878174, 36.82747268676758, 2.519317865371704, 17.8096981048584, -26.816518783569336, 30.02869415283203, -19.923595428466797, 29.971345901489258, 2.01529598236084, -29.55894660949707, 47.97972106933594, -27.207429885864258, -7.720160961151123, -24.25005340576172, 59.836551666259766, 5.8423662185668945, 15.355217933654785, 16.5395565032959, -7.977106094360352, -7.430368900299072, 54.4855842590332, -21.677797317504883, 59.280277252197266, -42.38480758666992, 6.115250110626221, 3.724479913711548, 14.038992881774902, -0.7836441993713379, -10.31616497039795, 13.578043937683105, -13.163623809814453, -35.78388977050781, -11.311519622802734, 12.589875221252441, 2.5058083534240723, 0.3330908417701721, 21.53177833557129, -30.861879348754883, 4.3424177169799805, -8.191526412963867, 6.580587863922119, -29.5977840423584, 18.01863670349121, 15.19520378112793, -2.150968074798584, -51.633384704589844, 69.7929916381836, -51.148983001708984, 16.08070945739746, 5.097894668579102, -24.441875457763672, -4.587555408477783, -28.246109008789062, -3.4223599433898926, 13.313170433044434, -6.867416858673096, -18.012372970581055, -6.2241363525390625, 37.740806579589844, -53.845760345458984, -37.81574249267578, 15.813016891479492, -51.69297790527344, -7.788150787353516, -12.324386596679688, 40.26681137084961, -11.348404884338379, -7.104867935180664, 14.2315034866333, 15.171481132507324, -55.175296783447266, 18.942739486694336, -10.142385482788086, 45.44111633300781, -12.846691131591797, 15.896562576293945, 31.884946823120117, -28.47172737121582, 25.122453689575195, -16.832643508911133, 51.43756866455078, -15.164446830749512, 42.932621002197266, 23.64019203186035, -17.771203994750977, -22.53974151611328, -6.24542760848999, 37.44232177734375, -17.083955764770508, 3.099405288696289, 10.543885231018066, -13.714302062988281, 13.050718307495117, 67.5047607421875, -4.604223728179932, 49.083683013916016, 28.086376190185547, 42.31262969970703, -3.275263786315918, 2.1507821083068848, -36.88102340698242, 21.212697982788086, -1.0356431007385254, 20.799924850463867, 8.551156997680664, 37.54155349731445, 19.498388290405273, -0.8266986608505249, 34.516361236572266, -39.13686752319336, 42.70392608642578, 0.7647227644920349, -47.96550369262695, -7.912386894226074, 28.228612899780273, -18.546894073486328, -6.759160041809082, 29.416305541992188, 81.3328628540039, 18.42397117614746, 44.0421257019043, 54.504024505615234, -65.94015502929688, -33.242496490478516, 9.436439514160156, -55.46656799316406, -50.71372985839844, 35.40304946899414, -25.70918846130371, 14.919381141662598, 8.749354362487793, -4.321010589599609, 2.4977691173553467, -10.12083911895752, -51.532928466796875, -42.6310920715332, 42.77936553955078, -11.02579116821289, 20.387229919433594, 16.35497283935547, 19.6264705657959, 21.025650024414062, 59.25994110107422, 30.02495574951172, -3.8842082023620605, 44.91964340209961, 2.930230140686035, -12.00753402709961, -11.41553020477295, -0.8885176181793213, 51.28000259399414, 21.169017791748047, 47.7044563293457, 52.55193328857422, -45.26072311401367, 29.82284164428711, -8.040706634521484, -32.77754592895508, -15.878379821777344, 32.27204895019531, -4.567509651184082, -12.477076530456543, -30.882930755615234, -17.81922721862793, 30.483430862426758, -43.52219009399414, -5.109337329864502, -10.708559036254883, 10.272269248962402, -38.278831481933594, -19.118911743164062, -20.579200744628906, 2.8438994884490967, 25.38005256652832, 14.016430854797363, -59.45595932006836, -1.2417707443237305, 22.31878089904785, -18.864988327026367, -22.279064178466797, -36.74853515625, 13.609929084777832, -12.309285163879395, -76.25323486328125, -29.54698371887207, 3.039212942123413, 33.76415252685547, -10.31761360168457, -31.081443786621094, 43.197845458984375, -25.088356018066406, 40.88682556152344, 2.546959400177002, 2.049290657043457, -32.506649017333984, 2.3690671920776367, 31.498157501220703, -24.911142349243164, 2.1357123851776123, -7.056239128112793, -12.316608428955078, -28.062986373901367, -3.699201822280884, -14.003740310668945, 19.563232421875, -19.11532211303711, -2.7552671432495117, 14.427236557006836, -3.3669395446777344, -35.69227600097656, -40.22609329223633, -33.790611267089844, 10.805994987487793, -29.880992889404297, -38.20893478393555, 66.91239166259766, 18.76301383972168, 34.985408782958984, -67.51876068115234, -28.6779842376709, -5.7352681159973145, -24.64065933227539, 16.828962326049805, 12.505178451538086, 14.098641395568848, 35.21823501586914, -13.372879028320312, 13.2761812210083, 39.020263671875, 6.773130416870117, 7.575209617614746, -9.357118606567383, 22.700960159301758, -7.156735897064209, 19.348407745361328, -11.462217330932617, -22.06307601928711, 54.2342529296875, 1.9378973245620728, -11.780685424804688, -39.4185791015625, -21.226377487182617, 62.118831634521484, 37.6512336730957, 34.65892028808594, 10.023730278015137, 3.0302462577819824, 54.853790283203125, -28.462512969970703, 5.657744884490967, 20.28315544128418, -24.828367233276367, -27.668132781982422, -4.017760276794434, 15.039113998413086, -11.762978553771973, 13.501312255859375, -51.818904876708984, 29.286144256591797, -8.981558799743652, -16.98560333251953, 58.757694244384766, -7.193427085876465, -19.214557647705078, 22.480234146118164, -7.073574066162109, -37.76601791381836, -15.590311050415039, -15.25849437713623, 12.482471466064453, 0.24652302265167236, -62.39998245239258, -0.5273594856262207, 22.27791976928711, 0.7707301378250122, -45.292449951171875, -20.7664852142334, -0.8909049034118652, -17.149682998657227, 41.07646942138672, 2.77583646774292, -9.234786987304688, -35.02676010131836, -5.817626476287842, -13.803881645202637, 26.34429359436035, 23.275936126708984, -30.888681411743164, 33.39971160888672, -9.364317893981934, 27.658851623535156, 11.252471923828125, -2.5687220096588135, -0.5115056037902832, -23.648818969726562, -8.635665893554688, 24.436012268066406, -25.44329071044922, 3.463613986968994, -34.90022277832031, -31.542051315307617, -23.913372039794922, -60.86553192138672, 26.24114227294922, 8.137236595153809, -6.426713466644287, 9.677630424499512, 37.404014587402344, -16.46510124206543, -46.19384765625, -12.371501922607422, -5.565181732177734, 3.137075424194336, -3.753539800643921, -8.719321250915527, -76.22966003417969, -32.113014221191406, -4.3344011306762695, -24.41147804260254, 72.29611206054688, 43.62351989746094, 16.63077163696289, 2.413700819015503, 34.05219268798828, 4.55661678314209, 15.36799430847168, -6.497952938079834, 11.142426490783691, -14.617812156677246, -9.622264862060547, 65.09215545654297, -0.17676115036010742, -2.3865339756011963, -19.379247665405273, 43.49180221557617, -33.842891693115234, 21.618619918823242, -6.366735458374023, 5.425472259521484, -21.662153244018555, -0.8283947110176086, 15.660656929016113, -1.4942646026611328, -8.080558776855469, -24.77374267578125, 8.593768119812012, -29.986284255981445, 9.960860252380371, 35.73133850097656, 21.740694046020508, -25.81454086303711, 1.315528392791748, -18.495542526245117, 14.998974800109863, 16.343175888061523, -48.97517013549805, 1.3848965167999268, -20.377361297607422, -22.863595962524414, -25.456581115722656, 15.288394927978516, -0.21720290184020996, 6.608253479003906, 17.700210571289062, -6.974689960479736, -52.06355667114258, 16.8447208404541, -22.547250747680664, 17.440706253051758, -20.904678344726562, -6.533376216888428, -48.751686096191406, 53.64705276489258, 7.8758463859558105, -25.467395782470703, -11.610468864440918, -18.267812728881836, 16.99981689453125, -31.010778427124023, -9.933086395263672, 22.995576858520508, -28.312406539916992, 27.293743133544922, -0.8096125721931458, 64.30049133300781, 11.853645324707031, 20.166208267211914, -13.92590045928955, -9.115538597106934, 2.9384517669677734, -27.509387969970703, -39.60045623779297, -29.279367446899414, -2.1224312782287598, -12.070079803466797, -32.64387130737305, 7.333272457122803, -0.1866319179534912, 15.128552436828613, -36.84333038330078, -5.283358097076416, -12.51323127746582, -21.298702239990234, 1.2110562324523926, 35.30997848510742, 1.6724975109100342, 8.159693717956543, 49.53084945678711, -27.894527435302734, 22.50104522705078, -27.45952796936035, 25.841175079345703, -46.06459426879883, 27.767391204833984, 2.3889946937561035, 17.724136352539062, -18.83399772644043, -35.42806625366211, -80.66907501220703, -7.781935214996338, -39.235130310058594, -22.0500545501709, 9.669211387634277, -35.066810607910156, -0.07941985130310059, -12.487869262695312, 18.176639556884766, 21.767539978027344, 42.323341369628906, -25.197898864746094, -5.4278388023376465, 44.6983757019043, 27.976974487304688, 28.32870864868164, 12.382984161376953, 41.019248962402344, -15.246147155761719, 5.940217018127441, 48.631561279296875, -52.60603713989258, -24.177248001098633, 27.88253402709961, -1.6676948070526123, -39.79405975341797, -47.44245147705078, 17.357009887695312, -24.663959503173828, -15.668438911437988, 3.458496570587158, 7.892411231994629, -17.812788009643555, -91.63235473632812, -4.159976005554199, 35.14730453491211, -36.514862060546875, -17.35418701171875, 18.946311950683594, 2.1125214099884033, 4.993213176727295, 0.8667095899581909, 48.50288772583008, -17.200538635253906, -10.476914405822754, 49.561119079589844, 51.61856460571289, 28.55793571472168, 32.32266616821289, -31.38297462463379, -0.2823382019996643, -26.927038192749023, 42.23337173461914, 5.485747337341309, -30.561025619506836, -62.42667770385742, -15.90822982788086, 43.76670837402344, -10.45694637298584, -12.914422988891602, -46.14133834838867, -16.893516540527344, -75.26387023925781, 1.8436534404754639, -0.1688159704208374, -3.94779634475708, 16.89627456665039, 22.91107749938965, -63.510101318359375, -16.015625, -47.87841796875, -54.91974639892578, 22.0865421295166, 12.113883018493652, -20.760154724121094, -38.46144104003906, -6.38761043548584, -3.758263111114502, -20.787771224975586, 55.66877746582031, -1.6970272064208984, -42.352481842041016, 57.87181091308594, -19.227399826049805, -40.9013786315918, 9.709675788879395, -19.808076858520508, 8.195457458496094, -34.25423812866211, 27.690380096435547, -23.59721565246582, -45.34394836425781, 46.0865592956543, -35.053802490234375, 66.8602294921875, 35.59598159790039, -12.69627857208252, -34.8030891418457, 8.578787803649902, -20.434009552001953, -24.243831634521484, 2.222928047180176, -45.16120147705078, -30.968114852905273, 6.075485706329346, 24.28436279296875, -11.65589714050293, 0.28469210863113403, 14.082053184509277, 22.450756072998047, -27.658639907836914, 34.189109802246094, -36.571964263916016, -55.39685821533203, -0.11429333686828613, -44.229820251464844, -25.232009887695312, 5.153189182281494, 24.90225601196289, -19.23533058166504, -1.281506896018982, 10.526578903198242, 4.921323776245117, -8.961258888244629, -15.007810592651367, -1.1806230545043945, -8.219989776611328, -30.991907119750977, -24.76892852783203, -46.287296295166016, -24.561050415039062, -7.8271708488464355, -26.62580680847168, -11.242172241210938, 33.23362731933594, 51.954254150390625, -99.45989990234375, 7.77502965927124, 4.9285197257995605, -8.461618423461914, 41.01544189453125, 40.04550552368164, -22.142742156982422, -3.123544692993164, 1.7452514171600342, -70.69017791748047, -7.523379802703857, 7.450230598449707, -21.125932693481445, 16.284900665283203, 5.63323974609375, 51.08565139770508, -4.060186862945557, -17.072338104248047, -2.0195844173431396, 0.0026480555534362793, 24.04819107055664, 21.992572784423828, -30.536317825317383, 44.31956100463867, 26.2991886138916, -63.51514434814453, 11.23029613494873, 21.572383880615234, -3.6972460746765137, -26.413232803344727, -43.86467742919922, 14.168875694274902, 30.8995304107666, -2.183171033859253, 23.53243064880371, 70.31500244140625, -10.383161544799805, 34.298675537109375, 19.485252380371094, 23.975933074951172, -28.713468551635742, -13.751043319702148, 15.666220664978027, 7.430010795593262, -12.254119873046875, -5.110085487365723, 50.909446716308594, 39.73564529418945, -4.951766014099121, 19.720266342163086, -19.720781326293945, 12.79163932800293, -26.605228424072266, -3.01729679107666, 24.506946563720703, -0.40645384788513184, 6.343361854553223, 25.81305503845215, -12.591684341430664, 12.612014770507812, -18.71964454650879, 2.61483097076416, 32.38745880126953, 0.4095956087112427, -28.21944808959961, 23.645458221435547, 27.28509521484375, -1.9552586078643799, -12.945157051086426, -7.118008136749268, -7.720197677612305, -2.217028856277466, -26.354429244995117, 4.168548583984375, -0.7961217164993286, 7.4135355949401855, 35.00092697143555, -8.919361114501953, 33.507205963134766, -32.03857421875, -34.37339782714844, -3.144425868988037, 13.639904975891113, 7.924637317657471, 33.387062072753906, 6.125190734863281, 34.03527069091797, 51.408103942871094, 3.3035154342651367, 5.002785682678223, -34.28054428100586, -36.23543930053711, 23.444799423217773, 47.4953498840332, 33.01827621459961, -3.5193560123443604, 13.796971321105957, 17.651485443115234, 40.61001968383789, 24.625497817993164, -39.886695861816406, 39.7763557434082, -32.40103530883789, -0.6881372928619385, -20.362951278686523, -26.132993698120117, 16.09967041015625, -55.61126708984375, -16.84006690979004, -9.415387153625488, 3.872814893722534, 24.14120101928711, -3.629478931427002, 15.978147506713867, -36.018165588378906, -17.462411880493164, 21.98219871520996, 6.725205898284912, 39.311527252197266, 30.965944290161133, -2.3189048767089844, -45.54261016845703, 4.455033779144287, -51.491703033447266, 20.14209747314453, 18.13119125366211, 15.180282592773438, -17.443115234375, -14.643732070922852, 35.100276947021484, 24.39426612854004, 29.81262969970703, 25.60321807861328, 54.7021484375, -6.835472106933594, 0.6775780916213989, -3.818638563156128, 8.907209396362305, 3.1323020458221436, -4.876267910003662, 16.968170166015625, -11.6513090133667, -1.519652247428894, -7.987308502197266, -13.454029083251953, 17.119577407836914, -5.016120910644531, 36.8331413269043, -6.454588890075684, 22.826528549194336, -80.21566009521484, 1.8342633247375488, 0.11384981870651245, -10.892723083496094, 34.97746658325195, 12.091228485107422, 26.268037796020508, 4.115988254547119, -26.420021057128906, 5.6341233253479, 7.108361721038818, 37.35585403442383, 47.36009979248047, 23.55620574951172, -10.811368942260742, -86.95013427734375, 44.3515625, 50.431034088134766, -42.184356689453125, -33.4034309387207, -14.19806957244873, -9.09427261352539, 6.464229583740234, -40.084712982177734, 2.5474414825439453, 29.231571197509766, -14.90562629699707, 23.2221736907959, -12.849384307861328, -12.663939476013184, 67.09535217285156, -60.30076217651367, 30.776609420776367, -58.78972244262695, -33.85372543334961, -29.91117286682129, -22.245849609375, 0.9402247667312622, 42.15253448486328, 7.814047813415527, 51.14918518066406, 4.954491138458252, -21.713367462158203, 23.05425262451172, 33.411956787109375, 30.912673950195312, -3.047318696975708, 16.457984924316406, 33.940521240234375, -30.119781494140625, 19.30469512939453, 35.753334045410156, 2.1111228466033936, -35.88905334472656, 8.371625900268555, 34.03399658203125, -23.58062171936035, -29.662538528442383, -45.2340087890625, 5.611474514007568, 1.2964967489242554, 60.290916442871094, -1.6204099655151367, 27.85531234741211, -17.211559295654297, -11.88060188293457, 26.08186912536621, 12.966526985168457, 48.904510498046875, -0.6934471130371094, -23.27214813232422, -34.984580993652344, -12.859207153320312, -16.700939178466797, -31.534496307373047, 39.81733703613281, 71.23231506347656, -31.700881958007812, -8.59887981414795, -91.00736999511719, 23.927474975585938, 19.061477661132812, 33.206485748291016, -4.4944047927856445, 32.19083023071289, -6.0826544761657715, -44.2046012878418, 42.873348236083984, 33.55470657348633, -28.339519500732422, -17.582530975341797, 47.18272018432617, -44.95769119262695, -32.92585754394531, 5.820054531097412, 10.93968677520752, 40.24711990356445, 0.05798661708831787, -10.02208137512207, -4.898209571838379, 32.083980560302734, 23.688589096069336, -9.591992378234863, -59.87675094604492, -16.524354934692383, 9.824275970458984, -48.79292678833008, 57.28251647949219, -28.545433044433594, 17.079816818237305, 6.876192092895508, -18.151371002197266, 54.7034797668457, -10.370003700256348, 33.55498123168945, 1.5094761848449707, -23.387313842773438, -0.020992279052734375, -15.806954383850098, -19.17499351501465, 17.308149337768555, 8.02971076965332, 24.5195255279541, -3.5307610034942627, -45.46482849121094, 8.682808876037598, 28.396085739135742, -7.297905445098877, 33.37781524658203, -7.9006829261779785, 9.239977836608887, -5.177032947540283, -0.0689464807510376, 3.2476634979248047, 9.490477561950684, -2.195350170135498, -4.571957588195801, -18.243955612182617, -45.76144027709961, -35.81629180908203, -2.0799198150634766, -92.95954895019531, 5.214128494262695, 52.01911926269531, 40.33804702758789, 24.195070266723633, 18.94232749938965, -22.379940032958984, 7.645487308502197, -23.857120513916016, -8.848567008972168, 14.657341957092285, 39.57037353515625, 6.5036940574646, 32.422550201416016, -16.869953155517578, -49.75743865966797, 9.727153778076172, -30.602397918701172, 29.556217193603516, -39.69249725341797, -2.2043638229370117, 33.9952392578125, 15.256516456604004, 23.203166961669922, 54.55904769897461, 34.421485900878906, 56.572696685791016, -13.976903915405273, 22.859907150268555, -1.2850368022918701, -10.923089981079102, 46.6685791015625, -48.3548469543457, 34.0756721496582, 36.29356384277344, -4.508978843688965, 0.2005406618118286, -0.29704225063323975, 14.108335494995117, -43.1768798828125, 12.876176834106445, 9.747268676757812, -32.31106185913086, -1.6166969537734985, -10.339553833007812, -19.014720916748047, -21.646772384643555, -11.488459587097168, 20.80103302001953, -26.349340438842773, 4.026953220367432, 43.97905349731445, -33.70720291137695, 3.4289498329162598, 26.961170196533203, -29.967103958129883, -11.103350639343262, -25.15791893005371, -31.47970199584961, -42.5854606628418, -7.65168571472168, -11.053852081298828, -66.48723602294922, 7.217033386230469, 46.43815231323242, 36.6218147277832, -48.999786376953125, -4.077768325805664, -53.04934310913086, 11.116856575012207, -49.35439682006836, 8.875724792480469, -41.9893913269043, -58.60928726196289, -113.3332748413086, -12.613587379455566, -9.647971153259277, 22.429973602294922, -29.436588287353516, 17.353145599365234, -10.725135803222656, -30.828977584838867, -4.889150142669678, -33.29616928100586, 34.56467056274414, -11.806962966918945, 35.04657745361328, 48.40625, -2.1084847450256348, 18.30867576599121, -23.7172908782959, 17.81717872619629, 47.18780517578125, 18.305391311645508, -16.949058532714844, 25.436567306518555, 33.849510192871094, 6.06486701965332, -2.4119296073913574, -67.7314453125, 2.4338691234588623, 27.597898483276367, 19.343963623046875, -2.1169168949127197, -13.45727252960205, -52.726287841796875, 40.8900260925293, 16.889802932739258, 4.689177513122559, -11.379337310791016, -1.102121114730835, 38.5316276550293, 4.438382625579834, -11.813116073608398, -36.632259368896484, -8.111324310302734, 44.54689407348633, -27.85423469543457, -18.147441864013672, 41.17241287231445, -16.646350860595703, 9.382794380187988, 12.364814758300781, -18.76313018798828, 19.63364028930664, -25.82776641845703, 27.607086181640625, -30.81216049194336, 16.85356903076172, 7.363205432891846, -31.209716796875, -9.661052703857422, -28.040374755859375, -11.1934814453125, -28.9520320892334, -5.956715106964111, 23.30416488647461, 16.715116500854492, -9.572474479675293, 42.93696212768555, 25.872587203979492, 5.673084259033203, -26.03848648071289, 37.71916961669922, 9.863717079162598, -9.202007293701172, 25.381610870361328, -28.539709091186523, -19.095623016357422, 38.50711441040039, 31.57660484313965, -54.9566764831543, 33.31591796875, 5.386850357055664, -71.30321502685547, -6.270495414733887, -21.186481475830078, -3.8815834522247314, -86.07366943359375, -8.359720230102539, 18.831920623779297, -16.265243530273438, -6.172893047332764, -10.56812858581543, 36.770240783691406, 37.57039260864258, -25.06047248840332, -12.323312759399414, -49.37456512451172, 32.05165100097656, -44.87936782836914, 10.346875190734863, -15.045135498046875, -1.479530692100525, 40.92634963989258, 25.675159454345703, -13.804600715637207, -8.542859077453613, 6.337263107299805, -16.576993942260742, -51.793888092041016, -9.321493148803711, 27.795602798461914, 48.5998420715332, -24.036022186279297, 3.4059929847717285, -5.94592809677124, -2.7085039615631104, -12.322059631347656, 15.717499732971191, 2.715088367462158, 38.516319274902344, 1.089040756225586, -1.120789647102356, 31.95354461669922, -56.59056854248047, 7.153990745544434, -0.963342547416687, 32.96843338012695, 4.985435485839844, -15.891717910766602, 25.099140167236328, 7.917994499206543, -23.53641128540039, 5.425082206726074, 58.59916305541992, -60.527435302734375, 45.23002624511719, -30.80027961730957, -25.648698806762695, -5.0841064453125, 41.87661361694336, 11.565670013427734, 20.338762283325195, -3.518019199371338, 22.10589599609375, 28.250240325927734, -12.939577102661133, 17.028587341308594, 8.345746040344238, 35.76988220214844, -4.284205436706543, 41.49919509887695, 20.840641021728516, 7.918578147888184, -15.483428955078125, 26.218130111694336, 24.86471176147461, 21.09320831298828, 5.7387824058532715, 2.2232909202575684, -1.4636151790618896, -26.910629272460938, 16.97372817993164, 5.621189594268799, 56.34623336791992, -0.07212865352630615, -17.410537719726562, 54.787532806396484, 4.767122268676758, 2.7949118614196777, 4.89885950088501, -40.27671813964844, 30.85120964050293, -50.78905487060547, -9.500225067138672, 14.333684921264648, 15.908268928527832, 13.1659574508667, 40.8471565246582, -38.41078567504883, 8.06643295288086, 8.831134796142578, -11.780726432800293, -10.81445026397705, 1.8613004684448242, -20.280986785888672, 19.514331817626953, -41.127159118652344, -17.952817916870117, -7.277654647827148, -43.58717346191406, -24.449859619140625, 46.61320877075195, 43.99931716918945, 20.38306427001953, 5.4522857666015625, 35.73305130004883, 34.96347427368164, -4.039286136627197, -9.067849159240723, 11.239898681640625, 53.255516052246094, -25.81629753112793, -11.761008262634277, -11.725326538085938, -21.45932388305664, -0.026455208659172058, -32.02524948120117, 18.40487289428711, -11.357665061950684, -6.093472480773926, -47.67700958251953, 33.46875, 46.63941955566406, 32.13658905029297, -29.779212951660156, -30.157066345214844, 17.113765716552734, 5.841508388519287, 35.24365997314453, 31.81181526184082, 3.3739013671875, 4.266456604003906, -24.67892074584961, 15.679476737976074, 8.259678840637207, 27.931615829467773, -5.491103172302246, 78.59469604492188, 16.35785675048828, 24.662567138671875, -13.006547927856445, 73.37605285644531, 18.227401733398438, -30.285430908203125, 7.267956256866455, -24.160722732543945, 15.385857582092285, -1.8322392702102661, 8.028034210205078, 10.660202980041504, -11.995000839233398, -3.084927797317505, 29.396766662597656, -2.789240598678589, 32.621604919433594, 46.317237854003906, 34.172035217285156, 15.070146560668945, 6.76103401184082, 25.250268936157227, -24.586532592773438, -20.99264144897461, -7.294371604919434, -21.48625946044922, 55.53439712524414, 5.188199043273926, -6.572877883911133, -17.074420928955078, -50.920562744140625, 15.948704719543457, 21.744165420532227, 13.142949104309082, -4.468080520629883, 51.92304611206055, 52.15036392211914, 4.7625017166137695, 40.86274719238281, 12.756672859191895, 1.226736307144165, -11.612715721130371, -1.2299033403396606, -28.07453727722168, 34.52515411376953, 26.310401916503906, -30.892566680908203, -18.01725959777832, 30.36739158630371, -74.14706420898438, -6.433823585510254, 44.73955154418945, -2.9922244548797607, 44.46332550048828, -26.72631072998047, -33.98966598510742, 10.798602104187012, 60.078514099121094, -12.734972953796387, 41.42448806762695, -18.53960418701172, -3.656113624572754, -58.24541473388672, -19.13849639892578, -4.867969512939453, -2.751277446746826, -18.246076583862305, 20.196197509765625, 28.03467559814453, 5.969381332397461, -37.279518127441406, 20.74872589111328, -6.929364204406738, -15.733098030090332, 27.86173439025879, -16.743736267089844, 22.988117218017578, 17.640167236328125, -16.802621841430664, -18.893665313720703, 2.4711484909057617, 20.94641876220703, 31.296600341796875, 60.46084976196289, -14.226978302001953, -14.345285415649414, 19.963977813720703, -22.90671157836914, -47.23028564453125, -46.80139923095703, 37.55491638183594, -15.261088371276855, -42.40670394897461, 100.10743713378906, 0.9733265042304993, -36.62629699707031, -7.00106143951416, 6.495203495025635, 3.38741397857666, -2.4815187454223633, -0.6507208347320557, 3.2965524196624756, -24.060747146606445, -98.95610809326172, 9.840831756591797, 43.41434097290039, -2.5495872497558594, -27.687885284423828, 22.704181671142578, 11.551558494567871, -2.0071072578430176, -13.483391761779785, -9.365775108337402, -3.909313201904297, -40.70406723022461, -40.123268127441406, 10.171895027160645, 13.940287590026855, 35.57511520385742, -28.548995971679688, -19.688112258911133, 4.14014196395874, -44.624752044677734, 18.056339263916016, -14.579524993896484, 18.856781005859375, -43.71955490112305, -11.035694122314453, 38.878623962402344, 17.463930130004883, -25.538610458374023, -9.854616165161133, -6.57297420501709, -34.538597106933594, -14.592962265014648, 49.18331527709961, 64.04389190673828, 9.717227935791016, 52.298240661621094, -26.82942008972168, -19.949905395507812, -5.785019397735596, -6.736696243286133, 1.1013782024383545, -11.308764457702637, -85.72306823730469, -13.813311576843262, 10.99450397491455, -10.867790222167969, 67.98822784423828, 18.880659103393555, 63.651893615722656, 0.06462180614471436, 12.184070587158203, 25.13264274597168, 10.934532165527344, 11.507309913635254, 10.279928207397461, -4.651460647583008, 8.051149368286133, 14.162519454956055, -17.41792106628418, 36.10688781738281, 2.4786086082458496, 42.72886276245117, 10.46795654296875, 30.88796043395996, 6.068619728088379, 50.402069091796875, 43.45134353637695, 18.97026824951172, 25.977636337280273, 41.50993728637695, -4.6113104820251465, 2.240638256072998, 12.088664054870605, -48.7005500793457, 16.113557815551758, -36.586395263671875, -38.67850875854492, -7.332001686096191, 35.85981369018555, -10.13212776184082, 24.474639892578125, 14.878589630126953, 22.87169075012207, -28.810916900634766, 54.43500518798828, -6.90005350112915, 50.37826919555664, 39.545196533203125, -48.67900085449219, 48.019840240478516, 16.607412338256836, -9.100076675415039, -64.09578704833984, 6.438485145568848, 52.452964782714844, 52.42449951171875, 5.416269302368164, 9.530417442321777, -21.695693969726562, -25.530147552490234, 10.855002403259277, -0.23271524906158447, -15.557788848876953, 1.7135026454925537, -56.83103561401367, 21.360963821411133, -52.14214324951172, -25.726337432861328, 1.1258974075317383, -19.434707641601562, -7.313683032989502, 24.78328514099121, 5.514005184173584, -13.236817359924316, 43.12394714355469, -8.888750076293945, -2.309037685394287, 32.83477783203125, -15.336431503295898, 26.02338218688965, -5.603504180908203, -25.131973266601562, -41.316585540771484, -52.73683547973633, -33.2746467590332, 19.431659698486328, 2.085829257965088, 4.434354305267334, 4.099399089813232, -14.187843322753906, 30.7114200592041, -9.021324157714844, -57.42656707763672, 12.799055099487305, 47.23273468017578, -15.003942489624023, 2.9619221687316895, 14.495851516723633, -28.74846076965332, -45.60667037963867, 3.3341729640960693, 35.29972457885742, 31.07722282409668, 4.812856674194336, -10.862295150756836, 11.982383728027344, 17.632219314575195, 24.49117088317871, 60.164817810058594, 5.842064380645752, -24.106433868408203, -3.2328662872314453, -38.84309387207031, -13.760477066040039, -0.6148264408111572, 3.0519957542419434, -10.750038146972656, 63.149391174316406, -20.61972999572754, 9.075105667114258, 97.80358123779297, -9.855088233947754, -27.962404251098633, -26.65077781677246, -15.992385864257812, 41.42137908935547, 1.930029034614563, -30.793943405151367, 0.2647593021392822, -21.8039608001709, -86.26484680175781, -45.31196212768555, -15.120712280273438, -41.72931671142578, 30.4077091217041, 16.310842514038086, -30.599637985229492, 48.847557067871094, -0.7792915105819702, 1.2537543773651123, 23.273258209228516, 16.783035278320312, 39.49465560913086, -8.061638832092285, -8.214195251464844, -17.62620735168457, 34.75775146484375, 37.866397857666016, -23.8885498046875, -43.90888595581055, -12.117727279663086, -55.82985305786133, -22.857078552246094, -23.01622772216797, -6.978010654449463, 38.66688919067383, 14.65616512298584, -57.85434341430664, -25.243871688842773, -33.95998001098633, -22.91877555847168, -27.76146697998047, -2.529421091079712, 15.821236610412598, 4.4096269607543945, 69.71769714355469, -8.530351638793945, 44.28071975708008, 3.753159761428833, 19.2873477935791, -46.81222152709961, 23.41147804260254, -28.914655685424805, -6.246347904205322, 11.803956985473633, 15.9551362991333, 27.137170791625977, 10.191657066345215, -40.03894805908203, -36.45741271972656, -17.632539749145508, 3.4253125190734863, -27.83673858642578, -8.862459182739258, 0.20045489072799683, -6.2633185386657715, 22.781476974487305, -28.22062873840332, -25.853439331054688, -30.984111785888672, 13.135385513305664, -17.751440048217773, -21.3793888092041, 20.026710510253906, 26.576126098632812, -28.56441307067871, 24.479827880859375, -36.04323959350586, 26.031877517700195, -19.510087966918945, 12.289031028747559, 9.71345043182373, 20.18622398376465, -3.494363307952881, -18.116107940673828, -5.489001274108887, 31.035120010375977, 0.9827443957328796, 31.436840057373047, 76.04176330566406, 16.270809173583984, 35.334564208984375, -3.1005849838256836, -8.191624641418457, -7.981288909912109, 17.212934494018555, 77.86937713623047, 4.163693428039551, -14.84815788269043, -13.63517951965332, -32.38853454589844, 25.169422149658203, -7.775853633880615, -60.245784759521484, 12.052090644836426, -60.7443733215332, 15.321231842041016, 29.844533920288086, 27.72565460205078, -20.51485252380371, 44.20216369628906, 46.820594787597656, 29.41828155517578, 8.457435607910156, -19.4556941986084, 51.36198043823242, 0.030204296112060547, -66.74873352050781, 6.236627578735352, 40.80378341674805, -2.1788530349731445, 14.773633003234863, -27.078943252563477, -4.010281562805176, 24.678367614746094, 2.99515962600708, 27.76645851135254, 7.530532360076904, 42.22633361816406, -49.215877532958984, -10.522480964660645, -38.643714904785156, 0.48216116428375244, -52.81828689575195, -28.55263328552246, -39.20954513549805, -38.14493942260742, 27.378372192382812, -44.528926849365234, -25.261289596557617, -27.05903434753418, 18.662853240966797, -27.904186248779297, 13.021544456481934, -18.299665451049805, 18.25858497619629, 21.137845993041992, 11.998071670532227, -47.338958740234375, 64.69833374023438, -52.589847564697266, 33.534053802490234, -18.45252799987793, 5.348980903625488, -61.714439392089844, 19.93436622619629, -28.57891845703125, 11.739401817321777, -2.3943049907684326, -15.792337417602539, -28.835128784179688, 10.218681335449219, 28.600238800048828, 28.174427032470703, 24.07623863220215, -33.46199035644531, 1.3496825695037842, -4.989748954772949, -1.402466893196106, -30.11246109008789, 16.107881546020508, -3.146340847015381, 31.18048667907715, -15.240724563598633, -41.18125534057617, -5.609064102172852, 28.661041259765625, 35.15858840942383, 28.60779571533203, -42.881717681884766, 25.05451202392578, 18.2278995513916, -44.06151580810547, 67.39006805419922, 11.744634628295898, 24.219013214111328, -48.810970306396484, -55.24999237060547, 30.707500457763672, -58.96922302246094, 47.56672668457031, 6.046750545501709, -10.515932083129883, -14.46597671508789, -71.28060913085938, 16.92923355102539, 13.048166275024414, -18.8421630859375, -20.641441345214844, 14.065725326538086, 26.164962768554688, 14.75251579284668, -18.388256072998047, -9.818229675292969, -46.650779724121094, 19.912790298461914, 24.118091583251953, 7.16690731048584, -13.54926872253418, 19.30542755126953, 0.08621340990066528, -15.478948593139648, 11.655158042907715, -0.6735020279884338, -56.135135650634766, 14.30517292022705, 27.237070083618164, 63.44500732421875, -45.66900634765625, -6.836889743804932, 62.03728485107422, 40.2070426940918, 38.058868408203125, 26.323057174682617, -5.666499137878418, -33.162200927734375, 31.070619583129883, 56.37369918823242, -47.87103271484375, -14.91256332397461, -19.89408302307129, 35.596885681152344, 21.867626190185547, -2.125741481781006, -52.34773635864258, 16.374988555908203, -0.08274318277835846, 1.2378095388412476, 3.655078649520874, -25.201416015625, 14.024589538574219, -43.11348342895508, -36.780494689941406, -7.467523097991943, -23.740882873535156, -3.94332218170166, -12.117026329040527, -54.52944564819336, 8.115355491638184, 13.405743598937988, -12.932534217834473, -17.0225772857666, 4.817135334014893, -73.57501983642578, -28.76291847229004, 3.298924446105957, 22.54323387145996, 11.10515022277832, -15.413780212402344, -4.4185991287231445, -6.961943626403809, -18.97825813293457, -50.3149299621582, -15.869319915771484, 21.30235481262207, 55.89262771606445, 40.398162841796875, 33.9282112121582, 19.47810173034668, 35.385398864746094, -23.359600067138672, -28.564441680908203, -28.216461181640625, -14.968406677246094, 7.213608264923096, 48.30862045288086, -8.578638076782227, 14.000910758972168, 21.40953826904297, 19.46023178100586, -18.43898582458496, -16.59687614440918, -4.978214740753174, 42.874778747558594, 22.729400634765625, 32.12245559692383, -61.861873626708984, -28.753738403320312, -3.1178510189056396, -28.25689125061035, 54.404056549072266, -1.514897346496582, 31.9270076751709, 33.58760070800781, 21.688541412353516, -6.261641502380371, 68.26599884033203, 29.7010498046875, -22.404010772705078, 13.598008155822754, -9.087961196899414, -6.803216457366943, -29.82758903503418, 10.679584503173828, 29.678287506103516, 28.542882919311523, 27.503393173217773, 13.151150703430176, -6.4666056632995605, -6.591974258422852, 2.147925853729248, 35.3873291015625, 0.6965069770812988, -41.013641357421875, -4.565728664398193, 2.9280314445495605, 10.673661231994629, -21.039077758789062, 68.96154022216797, 54.2755241394043, 65.41313934326172, 90.14765930175781, -17.88300895690918, 23.538692474365234, 16.2431640625, 15.760795593261719, -30.994699478149414, 18.620927810668945, 1.5737144947052002, -4.8845672607421875, 35.10387420654297, -12.018669128417969, -27.49338722229004, -24.64034080505371, -9.416862487792969, -0.8158640265464783, -53.47783279418945, 11.064596176147461, -25.716646194458008, 6.450565814971924, 33.34225845336914, -13.870018005371094, -34.52880859375, 21.231718063354492, 24.89301300048828, -43.69253158569336, 3.3629281520843506, -60.1922607421875, 0.5264132022857666, 5.781602382659912, -0.7446868419647217, 23.494922637939453, 10.909717559814453, 60.68116760253906, -15.243889808654785, 33.42422103881836, -2.7783846855163574, -19.744157791137695, -45.65890121459961, -47.160133361816406, -33.81801986694336, 4.8538031578063965, 36.694828033447266, 5.7017669677734375, 76.64240264892578, -0.03267550468444824, 33.6472282409668, 43.680335998535156, -13.006939888000488, -27.77876853942871, 7.171013355255127, 24.483463287353516, -26.007619857788086, 11.96035385131836, -52.96453094482422, -10.887083053588867, -3.1807351112365723, -76.09713745117188, 7.768398284912109, -40.36861038208008, -37.76618957519531, -8.139143943786621, 0.06636238098144531, -10.525541305541992, 32.49043655395508, -26.880422592163086, -33.445289611816406, -4.013270378112793, -50.19816970825195, -2.6534082889556885, -0.8861840963363647, 8.675300598144531, -51.11552810668945, -35.826175689697266, 11.910688400268555, 7.272138595581055, -55.845314025878906, -18.17814064025879, 4.8330302238464355, 18.137954711914062, 35.728782653808594, 13.156497955322266, -11.113885879516602, 18.402400970458984, 8.93150806427002, -11.503725051879883, 65.36639404296875, 13.648417472839355, 2.9904675483703613, -1.2453553676605225, 20.492263793945312, 1.4667497873306274, 76.98714447021484, 13.829840660095215, -40.86969757080078, 49.26482009887695, 2.419743537902832, 68.60202026367188, 25.858421325683594, 23.257272720336914, 1.5966765880584717, 40.92333984375, -10.400724411010742, -12.565109252929688, 48.43894577026367, 36.003055572509766, 4.678407669067383, 16.123977661132812, -46.05495834350586, 25.659732818603516, -4.807192802429199, 69.34980010986328, 10.273375511169434, 13.534674644470215, -5.426448345184326, -16.114625930786133, -6.159778118133545, 31.071922302246094, 3.1349568367004395, 17.33696746826172, 18.23918342590332, 9.786900520324707, 0.2258291244506836, 64.10945892333984, 8.128995895385742, 28.94190216064453, -16.942245483398438, 13.376338958740234, -15.494684219360352, 36.69025802612305, -6.292728424072266, 55.20352554321289, 9.817419052124023, -10.661346435546875, 13.239920616149902, -31.145835876464844, 23.335073471069336, 27.875816345214844, -30.439319610595703, 45.578407287597656, -0.738407552242279, -10.307060241699219, 10.006020545959473, -42.057106018066406, 0.2201905995607376, -1.2955753803253174, 4.801068305969238, 41.344825744628906, 2.285635232925415, -3.6995177268981934, 20.615800857543945, 25.94950294494629, -1.5811299085617065, -12.051714897155762, 2.6663379669189453, -36.89743423461914, 6.666342735290527, -50.896629333496094, 12.744379043579102, -9.855274200439453, -1.5421009063720703, 3.653639793395996, -11.545918464660645, 6.615312576293945, -78.46220397949219, 66.6611099243164, -45.99641799926758, 36.660789489746094, 36.56382751464844, 13.729501724243164, -0.9591952562332153, -4.5888471603393555, -10.375107765197754, -4.104081153869629, 21.59781265258789, 8.046076774597168, -24.562088012695312, 17.67012596130371, 35.5281867980957, -24.967082977294922, -7.292794704437256, -6.622031211853027, -41.190792083740234, 57.81272888183594, -51.62163162231445, 50.61198425292969, 5.732711315155029, -50.076324462890625, 35.7991828918457, 13.593340873718262, 56.37785720825195, 13.964000701904297, -5.986501693725586, -15.691264152526855, -17.19306182861328, -32.051239013671875, 10.502181053161621, -51.272438049316406, 90.07640075683594, -39.00041580200195, 45.75869369506836, -38.82603073120117, -36.29911422729492, 15.260733604431152, 4.367478370666504, 5.3092145919799805, 22.622543334960938, 49.11412811279297, 7.414353370666504, 24.56049919128418, 17.43388557434082, -7.968201160430908, 21.038293838500977, -8.072897911071777, -15.498916625976562, -33.9089241027832, 15.134605407714844, -48.56412887573242, 30.509552001953125, -15.44184398651123, 8.665876388549805, -9.551519393920898, 29.596290588378906, -20.842315673828125, 11.678190231323242, -51.61659622192383, 17.262784957885742, -23.180219650268555, -24.72757911682129, -6.934640884399414, -18.216876983642578, 48.71463394165039, 40.564151763916016, 32.34446334838867, -21.0994930267334, -21.13703155517578, -4.16206169128418, 22.70808982849121, -67.47608947753906, -19.032604217529297, -35.25014877319336, -24.991012573242188, -37.645118713378906, -45.392005920410156, -3.0792236328125, -35.734127044677734, -15.313172340393066, -14.236205101013184, 10.156838417053223, -59.47807693481445, 23.769357681274414, 7.140153408050537, -9.07625675201416, 31.02191925048828, 30.84622573852539, 30.590248107910156, 14.35280704498291, -40.002750396728516, -25.726404190063477, -28.67643928527832, 10.745654106140137, -42.768028259277344, -13.030267715454102, 41.551612854003906, 18.38734245300293, 48.01061248779297, -13.509891510009766, -56.1129035949707, 67.10474395751953, -10.985688209533691, 17.372539520263672, -15.393182754516602, 2.1675853729248047, 46.139060974121094, -54.1550178527832, 56.994544982910156, 2.82562518119812, 8.399412155151367, 25.618938446044922, 32.38353729248047, -11.386985778808594, -15.25551986694336, 5.454807281494141, -11.513758659362793, -9.48092269897461, -29.142356872558594, 17.32863998413086, 48.00312042236328, -7.272916316986084, -41.45357131958008, -44.52993392944336, 1.0287587642669678, -54.24878692626953, -12.356767654418945, -1.12209153175354, 15.694229125976562, 3.7455663681030273, -14.576910018920898, 23.771039962768555, 41.8284797668457, -14.586922645568848, 32.89405822753906, -7.086578369140625, 22.224897384643555, 33.74848556518555, 20.54466438293457, -20.510658264160156, 15.230286598205566, 11.07608413696289, 3.9913928508758545, -0.9060744047164917, -26.223691940307617, -43.705955505371094, -19.31559181213379, -44.037513732910156, 8.851996421813965, 26.953598022460938, -6.300350666046143, -27.466907501220703, 7.106855392456055, 10.531107902526855, -12.937535285949707, -79.173828125, 12.772626876831055, -14.836676597595215, 19.665231704711914, 19.220853805541992, -0.844109058380127, 0.33569300174713135, 31.517213821411133, -53.88111114501953, 0.154224693775177, 7.33946418762207, 27.65728759765625, -18.445186614990234, -31.63452911376953, 36.61146545410156, 50.07960510253906, -3.3808505535125732, 4.151121139526367, -34.69347381591797, -12.437950134277344, -1.9075111150741577, 0.20684775710105896, 48.936439514160156, -20.721729278564453, -21.59172248840332, -48.08561706542969, 36.353050231933594, 36.91082763671875, -6.215263366699219, 51.05841827392578, 9.38232707977295, 48.32632064819336, 12.504536628723145, 32.97844696044922, -34.95389938354492, -34.73667526245117, 23.218860626220703, 6.449365615844727, -16.350069046020508, 12.618468284606934, 1.94771409034729, 9.772357940673828, -8.534534454345703, 1.293243169784546, 10.004938125610352, 53.21155548095703, 11.690948486328125, 12.976810455322266, -49.911277770996094, 10.212373733520508, -39.48789596557617, 64.17620086669922, 0.6304535269737244, 24.74160385131836, 17.94306182861328, 10.621192932128906, -0.2285778522491455, 7.937871932983398, 13.216451644897461, 5.15899658203125, -8.396092414855957, -13.470566749572754, 13.771388053894043, 6.305551528930664, 5.707249641418457, 20.567251205444336, -6.916961193084717, -32.098148345947266, -2.5557780265808105, 17.840087890625, -16.315462112426758, -41.94985580444336, 23.330039978027344, 39.81204605102539, -7.887504577636719, 12.584029197692871, -69.46285247802734, -37.56227111816406, 4.245080947875977, 9.416594505310059, -54.86485290527344, -4.238711357116699, -15.241515159606934, -52.680763244628906, 25.497459411621094, -13.71223258972168, 19.930391311645508, 9.514164924621582, -23.88181495666504, 7.279659271240234, -20.694866180419922, 0.5548902750015259, 4.730256080627441, 39.10905456542969, 10.008621215820312, 25.98848533630371, 44.11166763305664, 18.5731258392334, -2.8468308448791504, -23.41242027282715, 57.20734405517578, -11.695012092590332, 51.225303649902344, -29.397253036499023, -21.404943466186523, 12.278721809387207, -39.428897857666016, 22.12680435180664, -80.62187194824219, -42.48490905761719, 47.41389465332031, 2.9383962154388428, 1.2386069297790527, 33.572208404541016, -23.258371353149414, -17.280506134033203, 23.288211822509766, -33.7894172668457, 18.680896759033203, 16.75933837890625, 60.56668472290039, 36.95733642578125, -4.2624053955078125, 2.5445101261138916, -0.825328528881073, -77.38728332519531, 26.458499908447266, 34.112525939941406, 34.04214096069336, -41.05198287963867, 38.39017105102539, -59.83798599243164, 2.6110451221466064, -10.93159008026123, 10.321569442749023, 39.41628646850586, 2.6979732513427734, 56.17510223388672, 37.66904830932617, -1.4878079891204834, 15.90516471862793, 27.575498580932617, -23.180971145629883, -31.253393173217773, -24.201122283935547, -4.340997219085693, 38.019309997558594, 47.54131317138672, 29.831518173217773, -9.492080688476562, 13.568052291870117, 34.632118225097656, -9.07339096069336, -40.92707824707031, -31.919044494628906, -8.805685997009277, -26.77671241760254, -22.40604591369629, 47.06526184082031, 18.529157638549805, 15.136445045471191, -37.40453338623047, -31.935382843017578, -32.33560562133789, 86.34545135498047, -20.570785522460938, -15.326518058776855, -17.2493839263916, -16.255725860595703, 5.8586106300354, -58.89064407348633, 2.2523303031921387, 4.587730884552002, 36.72557067871094, 31.627958297729492, -89.75820922851562, -18.900545120239258, 5.446227073669434, -42.848594665527344, 14.571124076843262, 0.16243243217468262, 0.28839394450187683, 25.78653335571289, 17.37644386291504, -9.250449180603027, 25.359046936035156, 5.195077896118164, 15.48274040222168, -17.983009338378906, 31.335142135620117, 0.5843649506568909, -0.6863275766372681, -60.82539367675781, -2.626676082611084, -39.23870849609375, -11.453609466552734, 25.93958854675293, 19.164209365844727, -1.3606395721435547, -36.912174224853516, 14.026636123657227, 8.595651626586914, -4.200887203216553, 70.77320861816406, -12.99167537689209, -14.42856502532959, -0.23046141862869263, -11.355055809020996, 5.778498649597168, 38.9563102722168, -0.016497701406478882, -20.60907745361328, 39.1411018371582, -13.40069580078125, -24.94193458557129, -12.922623634338379, 35.07154846191406, -39.68380355834961, -32.22367858886719, -22.277755737304688, -37.17421340942383, -27.11405372619629, -50.69795227050781, -2.9568121433258057, -56.99357604980469, 3.1152760982513428, -6.011578559875488, 2.7995545864105225, 25.814605712890625, -9.269587516784668, 18.734251022338867, -14.79209041595459, 21.68798065185547, -2.495389699935913, -13.261178016662598, -31.636489868164062, -53.99199295043945, -54.23788833618164, 73.82778930664062, 28.345062255859375, 9.083170890808105, 18.87481117248535, 9.317582130432129, -32.69084167480469, -35.55271530151367, 34.57266616821289, 28.720645904541016, -28.58001708984375, 18.4786376953125, 16.807374954223633, -42.026390075683594, -9.246747970581055, -23.341020584106445, -37.994319915771484, 24.36773109436035, -35.399742126464844, -12.200669288635254, 9.704669952392578, 27.936044692993164, -14.549278259277344, 21.676450729370117, 28.069080352783203, 26.93091583251953, -32.88933181762695, 36.8666877746582, 68.85107421875, 52.649261474609375, 10.05988883972168, -5.118139266967773, -25.292407989501953, 8.571145057678223, 59.3616828918457, -18.363191604614258, -39.21263885498047, -15.719868659973145, 7.674300193786621, 38.058082580566406, -24.815996170043945, 55.59617233276367, -53.098106384277344, -6.7144622802734375, -30.944530487060547, -28.984516143798828, -72.57503509521484, -28.86536407470703, 29.07874298095703, -31.141868591308594, -21.953020095825195, -32.229000091552734, -2.3415327072143555, -59.84994888305664, -3.6893091201782227, -6.922309398651123, -22.900325775146484, -34.89735794067383, 46.295589447021484, 24.86084747314453, 51.93336868286133, 34.83304214477539, -45.92913818359375, 12.958853721618652, 9.749574661254883, -20.06212615966797, 38.16257095336914, -12.498337745666504, -4.972166538238525, 13.750438690185547, 11.874898910522461, 55.47471237182617, 49.30961227416992, 6.374754428863525, 13.57482624053955, -13.782557487487793, -23.125247955322266, -15.78612232208252, 20.740131378173828, 8.654897689819336, 29.59389305114746, 15.835939407348633, -37.62716293334961, -27.919809341430664, 4.477644920349121, 1.8697768449783325, -0.11520460247993469, -12.902952194213867, -12.696002960205078, -11.800704002380371, 4.436445236206055, -4.11564826965332, -1.0820608139038086, -26.624441146850586, 7.56574821472168, 60.454307556152344, -20.166723251342773, -4.211635589599609, -9.477339744567871, -6.385458946228027, 45.74531555175781, -14.086341857910156, -23.842721939086914, -67.7486343383789, -54.80705642700195, -7.6464314460754395, -22.547584533691406, -21.921785354614258, 12.639001846313477, 15.729412078857422, -61.182613372802734, 14.606477737426758, 71.9893569946289, -23.333637237548828, -71.20252990722656, 14.538329124450684, -4.397228240966797, -37.90066146850586, 14.460731506347656, 9.401955604553223, 9.215353012084961, -1.6015393733978271, 28.62264633178711, -46.66737365722656, -38.167694091796875, -10.035079002380371, 10.056816101074219, 17.07986068725586, 20.470882415771484, -28.786481857299805, 24.952476501464844, -16.18463706970215, -45.45285415649414, 72.04322814941406, -26.654102325439453, 17.790390014648438, -4.844785690307617, 20.09307098388672, -9.509350776672363, -17.234149932861328, -9.141173362731934, 21.739877700805664, -96.6700439453125, -45.64868927001953, -33.13167190551758, -35.069854736328125, -15.518840789794922, 54.719791412353516, -15.17690658569336, 22.61708641052246, -36.854331970214844, -47.386898040771484, -44.82573318481445, -24.547182083129883, 17.956592559814453, 10.237570762634277, -2.4830689430236816, -20.951377868652344, 0.013625264167785645, -8.94645881652832, 18.5325927734375, 65.59285736083984, -28.967859268188477, -36.61741256713867, 9.359890937805176, -18.865123748779297, -58.84868240356445, 27.00701332092285, -46.23891830444336, 12.974187850952148, 38.47808837890625, -42.127960205078125, 5.639560699462891, 28.938766479492188, -13.783309936523438, -49.70246887207031, -16.161548614501953, 13.867249488830566, -31.707500457763672, -5.707441329956055, -7.209933280944824, -7.626275062561035, 3.457611083984375, 0.08562036603689194, 31.005878448486328, 14.540191650390625, -20.117271423339844, -50.71171569824219, -4.731403350830078, 15.869879722595215, -7.4285454750061035, 39.411415100097656, 47.482688903808594, -0.9324946403503418, 19.267614364624023, -46.111167907714844, 18.99153709411621, 24.100221633911133, -5.944451332092285, -22.339597702026367, 9.218379020690918, -48.220821380615234, -45.92658996582031, -26.571176528930664, 43.66526412963867, -20.731517791748047, -16.965167999267578, -72.92131042480469, 12.4789457321167, 32.436279296875, 18.66156578063965, 14.202384948730469, -11.831229209899902, 2.0710768699645996, 1.884434700012207, 6.1066813468933105, -17.247615814208984, 23.4787654876709, -33.01699447631836, -38.243553161621094, -0.6016770601272583, -7.065589427947998, -57.0056037902832, 26.527267456054688, 20.41027069091797, 2.477494239807129, -21.3502254486084, -6.839059829711914, -10.258913040161133, 24.963951110839844, 68.0903091430664, 44.411895751953125, -39.827423095703125, 28.2775936126709, -19.025325775146484, -5.990353107452393, 0.917748212814331, -24.329240798950195, 6.3770928382873535, 21.36819076538086, -25.46908950805664, -67.82501983642578, -33.86338424682617, -19.95519256591797, -12.766695976257324, -5.328076362609863, -2.6826579570770264, -55.51476287841797, -54.04450607299805, 26.481178283691406, -10.035937309265137, -5.1404500007629395, -63.31197738647461, -9.203112602233887, 9.165678977966309, -42.42884826660156, -11.614120483398438, 27.134183883666992, 14.953869819641113, 37.5974006652832, -31.765756607055664, 4.525067329406738, 1.5284967422485352, -4.975886821746826, -39.60868835449219, -0.13921388983726501, 12.940971374511719, 47.15391540527344, 12.499089241027832, 12.22763442993164, 15.084226608276367, -39.44403839111328, -23.700401306152344, -2.6056323051452637, -2.541300058364868, 27.614526748657227, 38.4899787902832, 21.219392776489258, -32.42656326293945, 20.061948776245117, 4.397757530212402, 63.265480041503906, -29.190027236938477, -4.5833048820495605, -5.439627170562744, 11.028396606445312, 94.49646759033203, 27.884262084960938, -21.79232406616211, -27.76947593688965, 3.8563995361328125, -11.479515075683594, 24.388803482055664, 4.112208843231201, -57.17923355102539, -14.527342796325684, -37.630943298339844, -40.155338287353516, -16.28888511657715, -33.94085693359375, 24.072458267211914, 15.437431335449219, 16.98143196105957, 12.851691246032715, -12.031683921813965, -56.898643493652344, 20.799758911132812, 30.909042358398438, 105.87545013427734, -9.066716194152832, -39.380859375, -7.621658802032471, 10.45071792602539, 10.127659797668457, 15.667009353637695, -78.17969512939453, -0.161103755235672, 6.1112847328186035, -52.864723205566406, 35.601741790771484, 16.221818923950195, -41.533451080322266, 12.318629264831543, 7.537720203399658, -3.133880853652954, -13.384403228759766, 1.5089375972747803, 43.41872787475586]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.04551880434155464, -0.38720497488975525, 1.8275554180145264, -0.03485862538218498, 0.21824684739112854, 0.2000025510787964, -0.41000568866729736, 1.4086241722106934, 0.5046241879463196, -0.40598905086517334, 1.2511813640594482, -0.7225990295410156, -0.04867381975054741, -0.6881620287895203, 1.2877485752105713, -0.4521103501319885, -0.7951824069023132, -0.5414422154426575, -0.45270416140556335, -0.9325507879257202, 0.913801908493042, -0.16985982656478882, 0.6811960339546204, -2.1867949962615967, 0.7387043833732605, -1.5248143672943115, -0.14223481714725494, 0.5312548875808716, -0.1488993614912033, 0.6340106725692749, -0.4276350140571594, 0.5684053897857666, 0.8932459354400635, -0.21707013249397278, 0.18572379648685455, 0.2409457266330719, -1.4784094095230103, 0.81252121925354, 0.4651385545730591, 0.6010962724685669, 2.8661611080169678, 0.8256785273551941, -2.6777896881103516, -1.23599374294281, -1.6609132289886475, -0.8074824810028076, -0.14647802710533142, 0.8447251915931702, -1.1801849603652954, -1.0927891731262207, 0.4942641258239746, -0.581311047077179, 0.46111443638801575, -0.1168687641620636, -1.8165456056594849, -0.827674925327301, 2.152012825012207, 0.4510197341442108, -0.17970407009124756, -0.712746262550354, -0.8911582827568054, -0.8146466612815857, 0.7724172472953796, 0.12442970275878906, -0.318233460187912, 0.6255196928977966, -0.7096537947654724, -0.4327540695667267, 0.4215949475765228, -0.45721203088760376, 1.792090892791748, -1.6971440315246582, -0.04089987277984619, -0.9736270308494568, 0.6661187410354614, -0.06118645891547203, 0.8022418022155762, 1.020909070968628, -0.22993244230747223, -2.0052521228790283, 0.7448137998580933, -0.7402285933494568, -0.6393601894378662, 0.2504312992095947, -1.6933252811431885, -0.36593908071517944, 1.1521763801574707, -0.17666271328926086, -1.2201982736587524, 1.3785728216171265, -1.433949589729309, 0.2681053876876831, -0.22934673726558685, 0.9249368906021118, -0.6575971245765686, 1.2052853107452393, -0.41079336404800415, -1.4469118118286133, 1.2094806432724, -2.370978832244873], [-0.6076205372810364, -0.8343160152435303, 0.7967577576637268, 1.3631685972213745, -1.0530450344085693, 2.121887683868408, 0.4538121521472931, 0.3755824565887451, 0.392927885055542, -2.090938091278076, -0.2976464331150055, 0.5792864561080933, 0.07851312309503555, 1.4495307207107544, 0.4964908957481384, 1.777001142501831, -0.2768109440803528, -0.29617583751678467, 0.1418948620557785, -1.695380449295044, 1.288046956062317, -0.24283170700073242, 0.6603875160217285, 0.3536651134490967, 1.9511675834655762, -0.023031918331980705, -1.5980955362319946, -0.9157567024230957, 0.13133065402507782, -0.9040283560752869, -0.5897586345672607, -1.7727657556533813, -0.44581499695777893, -0.12499231845140457, -0.08378338068723679, -0.7897660732269287, 0.2577957808971405, -1.7426903247833252, -1.2768092155456543, -0.2194565087556839, 0.9529587626457214, 1.1322455406188965, 0.6746366620063782, 2.0454905033111572, -0.20908571779727936, 1.3288792371749878, 0.10862007737159729, -0.867367684841156, -0.21246019005775452, 0.2993258535861969, -1.817694902420044, 0.3679516017436981, 0.7693865895271301, -1.348406434059143, -0.06372427940368652, -0.009121382609009743, 1.181530237197876, 2.1858067512512207, 0.6109066009521484, -0.20400728285312653, -0.6819273233413696, -0.619591236114502, 0.8449639678001404, 0.8114868998527527, -1.1016261577606201, 0.03843943774700165, 0.6309204697608948, 0.1720791459083557, -0.8116270303726196, 0.0880107581615448, 1.9429738521575928, -2.0547871589660645, -0.5885433554649353, 0.24725022912025452, -0.5846782922744751, 0.0985879898071289, -0.4664508104324341, -0.9357104301452637, -0.23401443660259247, 1.3482316732406616, 0.33060401678085327, 0.24100248515605927, 2.306199312210083, 0.9402550458908081, -1.4926213026046753, -0.578957736492157, 1.8657852411270142, -0.4820447266101837, 0.22593717277050018, -0.7583833932876587, -0.554611325263977, -1.6830166578292847, -1.7681339979171753, -1.1871819496154785, 0.10528327524662018, 2.5467214584350586, 0.16920672357082367, -0.3508546054363251, 0.9650647044181824, -0.5463487505912781], [-0.39126935601234436, 0.562964141368866, -0.4870857894420624, 0.6219422221183777, -1.1631959676742554, -0.7897711396217346, -1.4658279418945312, -0.9336022734642029, -0.12182430177927017, -0.2976509630680084, -0.9926219582557678, 1.138334035873413, -0.5754224061965942, -0.18749822676181793, -1.530339002609253, -0.22437801957130432, 0.6952292919158936, -0.3029947578907013, 0.2904627025127411, 2.8305180072784424, -0.0274796225130558, 0.03248751536011696, 0.08050484955310822, -0.39788034558296204, -0.7274386286735535, -1.2292673587799072, 1.8199161291122437, 1.4553937911987305, 2.0905940532684326, 0.783970832824707, 0.15976089239120483, -1.0247102975845337, -0.700714111328125, 0.5622242093086243, 1.4946290254592896, 0.17908941209316254, -0.9218160510063171, -0.9351518154144287, -0.4961335062980652, -0.09115726500749588, -0.007932918146252632, 0.8935789465904236, 0.005363216623663902, -1.4230257272720337, 0.41459307074546814, 0.276276558637619, 0.27881520986557007, 1.2979907989501953, -1.3570804595947266, 0.3112614154815674, 0.7328193187713623, 1.2545852661132812, 1.145810604095459, -0.34439918398857117, -1.2614707946777344, 1.4666492938995361, -0.1491333246231079, 1.554379940032959, -1.3652769327163696, 1.2462126016616821, -1.432093620300293, 0.5434487462043762, 0.45609211921691895, -1.1907342672348022, 0.8651296496391296, -0.3737366497516632, 0.1305919587612152, 0.33458423614501953, 1.542092204093933, 0.9203668236732483, 0.44818398356437683, 1.4663432836532593, 2.9146316051483154, -0.4908139705657959, -1.436549425125122, 1.0224361419677734, 0.04354174807667732, -1.6513817310333252, -0.5310470461845398, -0.21397829055786133, 1.3976191282272339, 1.3433970212936401, 0.5621232986450195, 1.4718773365020752, -1.3702442646026611, 0.36037755012512207, 1.7932511568069458, 0.8486778140068054, 1.0079278945922852, 0.6328962445259094, 1.8510284423828125, 0.19139036536216736, 1.6072593927383423, 0.08981679379940033, -1.5178457498550415, 0.02213749848306179, -0.7493635416030884, -0.09505435079336166, -1.0537837743759155, 0.007492630742490292], [-1.7055997848510742, -0.12336075305938721, 0.911027729511261, 0.7741685509681702, 1.523354172706604, -1.1211934089660645, -0.1446736752986908, -0.7299696207046509, -0.15634243190288544, -0.7059614062309265, -0.6867485642433167, 0.4109684228897095, -0.9869093298912048, 0.5851680040359497, -1.0079354047775269, 0.7074259519577026, -1.430422067642212, -0.7135044932365417, 2.4001526832580566, -0.6103618144989014, -0.5280840992927551, -0.9055519104003906, -0.8370832204818726, 1.713580846786499, -1.6113007068634033, -0.8652076721191406, -0.10110048204660416, -1.6317118406295776, 0.20177844166755676, 0.5811863541603088, 0.7768401503562927, -0.01811865158379078, 1.026930809020996, -0.26325300335884094, -0.012242686003446579, 1.4205670356750488, 0.4254189431667328, -0.47506195306777954, -1.5271967649459839, -0.04306716471910477, -1.9653736352920532, -0.06601788103580475, 0.37384337186813354, 1.1273524761199951, 0.5106409788131714, -0.6006150245666504, -1.4325509071350098, -1.2690260410308838, 1.890031099319458, -0.19636276364326477, 0.28435519337654114, 0.007066749967634678, -1.312658667564392, -1.255266785621643, 0.10222801566123962, 0.2551640570163727, 0.5890687108039856, 0.281693696975708, -0.6479759216308594, -0.4961240887641907, 0.3549784719944, 0.028002293780446053, 0.9398778676986694, 0.25480255484580994, 0.6274744272232056, 0.1654263138771057, 0.012347787618637085, 1.6391193866729736, 0.6994104385375977, -0.31296253204345703, -0.15937888622283936, 0.5751022696495056, 0.5493292212486267, 1.0077162981033325, 0.6961987614631653, -1.3713200092315674, 0.22535184025764465, 0.31375929713249207, 1.1287587881088257, -3.0795505046844482, 0.30704614520072937, -0.5335612297058105, -0.007364178542047739, 0.0008648941875435412, -0.5413207411766052, -0.681389570236206, 2.1135058403015137, 1.2523423433303833, -0.2962341904640198, 0.4650938808917999, 0.4601532518863678, -1.1350266933441162, -0.5617097616195679, -1.1343111991882324, -0.9028543829917908, 0.008914231322705746, 0.10377220809459686, -0.6098653674125671, 0.28623726963996887, 0.2511821389198303], [-0.3406231105327606, -0.46298256516456604, 0.6214037537574768, -0.05179256573319435, -2.1228320598602295, 0.19521787762641907, 0.25730833411216736, 1.1747119426727295, 2.4440860748291016, -0.3119942247867584, 1.867483377456665, -0.5586416721343994, 1.3785854578018188, -2.0847551822662354, 1.2133738994598389, 0.7164584398269653, 0.08557283878326416, -0.428854376077652, 0.18551109731197357, -0.8053920865058899, 1.7861510515213013, -0.7430951595306396, -0.9752386212348938, 0.31269845366477966, -0.8088867664337158, -0.03336816653609276, 0.15798988938331604, 0.3116595447063446, 0.6014958620071411, 0.675376296043396, 0.7556746006011963, -1.3776094913482666, -0.5593327283859253, -1.1541926860809326, 0.29432645440101624, -0.6542677879333496, 1.3723633289337158, 0.48098286986351013, 0.5145821571350098, -1.4813368320465088, 1.2856935262680054, 0.8018226623535156, -0.4973483085632324, 1.402127742767334, 0.07717127352952957, 0.05093834176659584, 0.6048048734664917, 1.1345034837722778, 0.7310310006141663, -0.476685106754303, 1.591618299484253, -0.3465261161327362, -0.7891480326652527, -0.2754140794277191, -1.2237780094146729, 0.11004285514354706, 0.4888857901096344, -0.09370426088571548, -0.9030904173851013, 0.4848438799381256, -1.131181240081787, -0.8390201330184937, -0.4851647913455963, 1.6371831893920898, -0.18139879405498505, -2.400437593460083, 1.2050223350524902, -0.8327209949493408, 1.2897483110427856, -0.22049933671951294, -1.7705518007278442, 0.47282296419143677, 0.7167167067527771, 0.008603799156844616, 1.0872479677200317, 1.242419958114624, -0.8632110953330994, 0.5890636444091797, 0.7909336686134338, -0.28006744384765625, 0.730614960193634, 0.5929543972015381, -0.03544138744473457, 0.7130228281021118, 0.4628550708293915, 0.06125136464834213, -0.40002351999282837, 0.4933713972568512, 0.296634703874588, 0.1084020733833313, 0.009758785367012024, -0.5930942296981812, -0.5644107460975647, 0.7557507753372192, 0.5310704112052917, -1.9312891960144043, 0.7392975687980652, 1.2740830183029175, -2.377941370010376, 0.3035445809364319]], "dtype": "float32"}, {"__nd__": [[1.0410630702972412, 0.28524866700172424, -0.5495805144309998, -1.092626690864563, 1.6139479875564575, -0.047598306089639664, 0.5490857362747192, -1.126672625541687, -0.5366984605789185, -0.4285239279270172, 1.1873598098754883, 0.6047837138175964, -1.6473017930984497, -1.1234426498413086, -0.7601739764213562, -0.6626198291778564, 0.5817486047744751, 2.2320199012756348, 0.9797161221504211, 0.41471853852272034, -0.08826714754104614, 3.3190674781799316, 0.12393202632665634, -1.767279863357544, 1.5219401121139526, 0.13333268463611603, 0.5288683772087097, -0.03831535577774048, 0.6041482090950012, 0.09130758047103882, -1.273276448249817, 0.014090967364609241, 1.0018287897109985, -1.6457208395004272, 1.0774052143096924, 0.4535219967365265, -1.3231885433197021, -2.6948351860046387, -0.19892515242099762, -0.9999696612358093, -1.4326565265655518, 1.678137183189392, -0.26497992873191833, -0.323803186416626, -0.8630964159965515, -0.8774043917655945, -1.0683337450027466, -0.11360006034374237, 0.6567262411117554, 0.7018430829048157, 0.7380889654159546, -0.7078408598899841, 0.0427187904715538, -0.9301521182060242, 1.0127930641174316, 0.8503131866455078, 1.8356672525405884, -0.5191602110862732, 0.2636966407299042, -0.6636260747909546, -0.2417706549167633, -0.2803865075111389, 0.9125432968139648, -0.43650245666503906, -0.36341074109077454, -1.07071053981781, -0.4735777974128723, -1.6253750324249268, 0.765487551689148, -0.5786470174789429, -0.7756897211074829, 0.8532587885856628, -1.4191652536392212, -0.15001358091831207, -1.50395929813385, -0.3073728084564209, 0.3905649781227112, 0.05114402249455452, -0.3343554139137268, -0.015930453315377235, -0.6840398907661438, 1.12252676486969, -0.5925054550170898, -0.17790654301643372, -0.39667394757270813, 1.5563486814498901, -2.149143695831299, -0.175333634018898, 0.5885013937950134, -0.7854838371276855, 0.38267210125923157, 0.9322802424430847, 0.9434356093406677, 0.32162901759147644, 1.7316137552261353, -0.08368393033742905, -1.634615182876587, -1.8466148376464844, 1.0065946578979492, 0.15326657891273499], [-0.14670412242412567, -0.8440958857536316, 0.32324647903442383, -0.42110130190849304, -0.8972918391227722, 0.19091801345348358, -0.062335047870874405, 0.45686522126197815, 1.0370420217514038, -0.8685598969459534, 1.149361491203308, -0.1301213651895523, -0.6937003135681152, 0.5657558441162109, -2.308380365371704, 0.05389046296477318, -0.4339815676212311, -0.4474075436592102, 0.4242599308490753, 2.66536545753479, -0.6688620448112488, -0.49916377663612366, -2.6296863555908203, 0.4976666569709778, 0.07931110262870789, -0.4924144148826599, -0.16599448025226593, -0.8071560859680176, 0.13281556963920593, 1.3494970798492432, -0.9484167695045471, 0.7901173830032349, -0.125246062874794, -0.6968995332717896, -0.3699782192707062, 0.6172386407852173, -0.03512971103191376, -0.4607384502887726, 0.014453809708356857, -1.144170880317688, -0.8364355564117432, 0.4792557954788208, 0.37857744097709656, 0.35339146852493286, -0.876679539680481, -0.5476091504096985, -1.1832623481750488, -0.7472107410430908, -0.724287748336792, -0.9891965985298157, 1.012477159500122, -0.7660658955574036, -1.0873112678527832, -0.2897379398345947, 0.6618385314941406, 1.238317847251892, 0.5901420712471008, -0.6313713788986206, 1.8472274541854858, -1.1593950986862183, 1.2795714139938354, 0.839999794960022, -0.5783630013465881, -0.008464711718261242, 0.9852831959724426, 0.03283093497157097, 1.4503737688064575, -1.7563868761062622, -1.2174066305160522, -2.873896360397339, -0.6265835165977478, -0.32950514554977417, 0.1977405697107315, 2.09132981300354, 1.2499293088912964, 0.9602281451225281, -1.0974693298339844, -0.797322154045105, -0.18182674050331116, -0.2812088429927826, 1.2338818311691284, -0.3177391588687897, 0.9149080514907837, 2.084033727645874, 0.3111865520477295, -1.2641048431396484, -1.158855676651001, -0.2355707734823227, 0.9563959240913391, -0.32866302132606506, -0.6217131018638611, 2.099804401397705, 1.263883113861084, -0.17748881876468658, -0.2377043217420578, -2.1353440284729004, -0.4538986384868622, -2.0602400302886963, 0.5055128931999207, 0.1471778154373169], [0.32272830605506897, -2.5286643505096436, -1.1963919401168823, 1.0237730741500854, 1.291229486465454, -0.06835980713367462, 0.05359702929854393, -0.7535755634307861, -0.5672927498817444, -0.16619113087654114, -0.322615385055542, -0.5565450191497803, -2.5240912437438965, 0.9367349743843079, -0.7450979351997375, 1.0952874422073364, 0.6970727443695068, 0.6577650904655457, -0.5811553001403809, -0.7734003067016602, 0.0417969673871994, 0.017777632921934128, -0.41315141320228577, 0.5820567607879639, 0.6071659326553345, -0.10308579355478287, -0.4420236051082611, -0.16106517612934113, -0.5411705374717712, -2.049592971801758, -0.6758694052696228, 0.8347640633583069, 0.2066020667552948, 0.695073664188385, 1.0206496715545654, -0.15428759157657623, 2.271137237548828, -0.8972694277763367, -0.4611639082431793, 1.963476300239563, 0.4170202612876892, 2.12740421295166, 0.5410168170928955, 0.21773843467235565, -2.260371446609497, -1.1595531702041626, 0.7910774946212769, -0.05674762278795242, 0.6624549031257629, 1.0822830200195312, -0.14379721879959106, 0.2514007091522217, 2.025423765182495, 1.2099443674087524, 0.07558245956897736, 0.3901212215423584, -0.093891941010952, 0.5373308062553406, -1.5392502546310425, 0.4587426483631134, 0.8995881080627441, 0.7065637111663818, 0.4915958344936371, -0.6970810890197754, -1.0776838064193726, -0.4097123444080353, -0.1077226772904396, 0.3803535997867584, 0.49966979026794434, 0.5843065977096558, -0.6537870764732361, 0.9152116775512695, -0.3837639391422272, 0.09190668165683746, 0.5148919820785522, 0.976815938949585, 1.0168061256408691, 1.0288920402526855, -0.7870703935623169, -0.22266365587711334, 0.2062181681394577, 1.3638278245925903, 0.24313168227672577, -1.0517842769622803, 1.2594947814941406, -0.07062473148107529, -0.5653942823410034, -0.7530588507652283, -0.7693097591400146, 2.754971504211426, -0.4079519808292389, 1.2821017503738403, -1.6565934419631958, 1.7251828908920288, -0.7656807899475098, 0.9156556129455566, -0.2776426672935486, 0.19371724128723145, -0.5829918384552002, 0.13786013424396515], [-0.32551291584968567, 0.9897719025611877, -1.2387560606002808, -1.5935603380203247, -1.3633224964141846, 1.769707441329956, -0.48339056968688965, 1.3879222869873047, 1.4972416162490845, 0.431487500667572, -0.9290075898170471, 1.0819395780563354, -1.2072179317474365, -2.1448590755462646, -0.26694411039352417, -0.1720414012670517, 0.7426322102546692, 1.4947106838226318, 0.01034857053309679, -0.1352851837873459, 2.906667470932007, 0.7484940886497498, 0.03496494144201279, -0.6295979619026184, 0.5367969274520874, -0.28684523701667786, 0.04929732903838158, 0.544918417930603, 0.7795103788375854, 0.606968879699707, 1.1558005809783936, 1.905192494392395, 0.4245283603668213, 0.5740146040916443, -0.8413485884666443, -0.3408620059490204, 2.153413772583008, -0.2866124212741852, 0.41315075755119324, -2.168841600418091, -1.2254395484924316, 0.8899524807929993, -0.05815901234745979, -0.6031656265258789, 0.2916666567325592, 0.21980610489845276, -0.5189955234527588, -0.28440624475479126, 0.10146596282720566, -0.3314955234527588, 0.17454837262630463, -0.3387208878993988, -0.2293764352798462, -1.1712125539779663, 0.8500531911849976, -0.7711929082870483, 1.2860183715820312, -1.3098313808441162, 1.2296819686889648, 0.02402685396373272, -2.970848321914673, -0.7458195090293884, -0.6382681131362915, 0.642794668674469, -0.20141170918941498, 0.4507143199443817, 0.2406177520751953, 0.4460010528564453, 1.2102221250534058, 0.8708131909370422, -0.20432503521442413, -2.460909605026245, -0.16427719593048096, -2.1073756217956543, 0.3536251187324524, 0.7820518612861633, 0.07032830268144608, -0.8727612495422363, -0.15718114376068115, -1.6182729005813599, -1.6920853853225708, 0.24944031238555908, 0.03650956228375435, 1.865684986114502, -0.7760869860649109, 1.3609575033187866, 0.06127122789621353, -0.18427090346813202, -0.05103301629424095, 0.01847047358751297, 1.272848129272461, 1.3181270360946655, -1.7602627277374268, 0.20019002258777618, -0.6835533380508423, -0.03810565546154976, 0.25316715240478516, -0.27430248260498047, 0.3725113868713379, 0.036501213908195496], [-0.9668756127357483, -1.1568337678909302, -0.466621458530426, 1.4562007188796997, 1.3359328508377075, -0.5544063448905945, 0.9181379675865173, -1.0712416172027588, 0.30791687965393066, 1.500550389289856, -0.3361574709415436, 0.8007700443267822, 0.4378439784049988, -0.9531310200691223, -1.0134700536727905, 1.7738555669784546, -0.707030177116394, -0.028534071519970894, 0.5339399576187134, -0.585306704044342, 0.4573908746242523, -0.44705769419670105, -1.377221941947937, 1.1402138471603394, 0.5376072525978088, -0.4809514582157135, -1.0001167058944702, -0.8866086602210999, -1.3555541038513184, 0.4867471754550934, 0.7626450657844543, -0.1523311883211136, -0.46808508038520813, 0.7766544818878174, 0.05888815596699715, -0.6668558716773987, 1.3313699960708618, -0.6952401399612427, 1.6567002534866333, 0.30288997292518616, 0.8163678050041199, 0.9851766228675842, 0.6929270029067993, -0.10270064324140549, 1.373086929321289, -1.2926831245422363, -1.1617323160171509, -1.9030506610870361, -0.648862361907959, -0.37936991453170776, -1.3206250667572021, 0.545525074005127, 1.264864206314087, -0.9782522320747375, 0.27092471718788147, -0.16048946976661682, 0.10478886216878891, -0.9327443242073059, 0.3835640251636505, -0.2483622282743454, -0.04023829847574234, 1.9483976364135742, 0.20353257656097412, -0.13874764740467072, -0.10389285534620285, 0.22383737564086914, -0.3558003008365631, -0.1704772561788559, 0.7860923409461975, -1.0606472492218018, 1.285895824432373, -0.15015988051891327, -0.8068161606788635, -0.8563673496246338, 0.1863478422164917, 0.38614293932914734, -1.7477688789367676, -0.8447474837303162, 0.8133516311645508, 0.9184842109680176, 0.5177733302116394, -0.3325243592262268, 1.0611335039138794, 1.1587193012237549, 0.4122338593006134, 0.3486778438091278, -0.2468721866607666, 0.9359237551689148, 0.4435844123363495, 0.11207978427410126, -0.7101995944976807, 0.5266913771629333, 1.2993431091308594, 0.10903000831604004, -0.7060380578041077, 0.5272882580757141, -1.036104440689087, 2.4827401638031006, 0.5397325754165649, 1.3177772760391235]], "dtype": "float32"}, {"__nd__": [-0.04980631172657013, -0.8707651495933533, -0.04412439465522766, 0.7367538809776306, 0.12885414063930511, 0.45398885011672974, -0.8143934607505798, 0.27025723457336426, 0.2091503143310547, 0.977921724319458, -1.2970603704452515, -0.2556142807006836, 0.2963903844356537, 0.9102939963340759, 0.5914616584777832, -0.31924062967300415, -0.10439079999923706, 0.0543711856007576, 1.0646209716796875, 0.7244191765785217, 0.350696325302124, 1.065028429031372, 2.011406421661377, -1.3244273662567139, 0.2846461534500122, 0.8125397562980652, 2.446341037750244, 0.5513784885406494, -0.875975489616394, -1.451320767402649, 0.932822048664093, -0.862943708896637, 1.4150562286376953, 1.4963886737823486, 0.3371905982494354, 0.8360967636108398, 0.6708974838256836, -0.34001684188842773, -0.1871524155139923, -0.35299986600875854, -0.7180490493774414, -0.5932114720344543, 0.1476241499185562, -0.5286975502967834, -0.15793515741825104, 0.7186341881752014, -0.6962472796440125, 1.0280098915100098, -0.5756271481513977, 0.10119805485010147, -0.24627919495105743, -1.5307871103286743, -1.0125612020492554, 0.020419957116246223, 1.2924221754074097, 0.1095358207821846, 0.3029087781906128, 0.5208311080932617, -1.290665626525879, 0.9048973917961121, -0.724242627620697, -0.13795745372772217, 0.4385521113872528, 0.0883953720331192, -0.16552166640758514, 1.8883134126663208, -0.14542871713638306, -1.3940234184265137, -0.5174911618232727, 1.7682814598083496, 1.788816213607788, -0.3261301815509796, 0.5836125612258911, -0.7352079153060913, -0.00930654164403677, -0.41905784606933594, 0.06903044134378433, 1.064805030822754, 1.6597949266433716, -0.04608886316418648, -1.3601881265640259, -2.2034804821014404, -0.7868178486824036, -0.4647568464279175, 0.040885500609874725, -0.6486817598342896, 0.14286987483501434, 1.0719645023345947, 0.7136399745941162, -1.2570997476577759, -0.35450103878974915, -0.962513267993927, -0.3464086651802063, -0.4430285096168518, -0.5333846211433411, -0.47531819343566895, -0.2924887537956238, 0.43802428245544434, -0.8581013679504395, 0.5515471696853638], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[-0.03449443727731705, 0.06176169961690903, -0.039228811860084534, -0.5778796672821045, 0.16423818469047546, 0.048133380711078644, -0.07879582792520523, 0.053009770810604095, -0.004666802939027548, -0.5677292346954346, -2.2003631591796875, 0.020950334146618843, -0.3496937155723572, -1.1472266912460327, 0.21707738935947418, 0.2475665658712387, 0.015499922446906567, 0.0639452189207077, 0.39031893014907837, -0.2609655261039734, 0.20140519738197327, 2.3332786560058594, 1.1265977621078491, 3.643150806427002, 0.4476531744003296, -0.7865497469902039, 0.6579920649528503, 0.1890808641910553, -0.27742454409599304, -0.7323119640350342, -1.1037853956222534, -0.34968721866607666, 1.8655381202697754, -1.9391554594039917, 0.29629701375961304, 0.40393614768981934, -1.3075734376907349, 0.4452424943447113, -0.03466007485985756, 0.09795209020376205, -0.7160729765892029, -1.0332751274108887, -0.30221620202064514, 0.5736928582191467, 0.2773153781890869, -0.8423309326171875, 0.5884053707122803, 0.5228692293167114, 0.20961762964725494, -0.02752283401787281, -0.2111382782459259, 1.372849702835083, -0.35490506887435913, -0.014873538166284561, -0.7226547598838806, 0.0017250565579161048, 0.8403036594390869, -0.02468915283679962, -0.07541517168283463, -0.8664417862892151, 0.5708090662956238, 0.10509366542100906, 0.514061450958252, -0.01919064112007618, 0.07849042862653732, -0.584822952747345, 0.11970824003219604, 1.9959367513656616, -0.42735418677330017, -1.274254560470581, 1.2648382186889648, 0.19146019220352173, -0.5927901268005371, 0.5747007131576538, 0.005424426402896643, 0.10744480788707733, 0.057281557470560074, 0.7941279411315918, -0.6515672206878662, 0.0648045688867569, -0.05750694125890732, -0.5860241651535034, 0.6742821335792542, -0.023448560386896133, -0.0594455748796463, -0.5371953248977661, -0.09908905625343323, -0.2624959945678711, -0.3136115074157715, -0.5186730623245239, 0.2592620849609375, -0.8037696480751038, -0.17208582162857056, -0.38419440388679504, -0.3985249996185303, -0.3708745837211609, 0.41619160771369934, -1.0036065578460693, -1.322900652885437, -0.8509262800216675], [0.027341868728399277, 1.0636166334152222, -0.035965319722890854, 0.5051144957542419, -0.18289150297641754, 0.7641345262527466, -0.23202039301395416, 0.1637267917394638, 0.21765615046024323, -2.106239080429077, -0.8039703965187073, -0.08355580270290375, -0.13269570469856262, 1.3350697755813599, -0.7799087762832642, -0.4253694713115692, 0.053999606519937515, -0.02942279912531376, 0.4386475384235382, 0.511376142501831, 0.15802910923957825, -0.5751060843467712, -2.882683515548706, -0.8205640912055969, 0.4206193685531616, -0.30479902029037476, -3.140676975250244, -0.691351056098938, -0.168392151594162, -0.47050705552101135, -1.0442155599594116, 0.6171150207519531, -0.588087260723114, -0.8950444459915161, -0.11134961247444153, -0.10497846454381943, 0.10871656239032745, 0.5452365279197693, 0.1719343364238739, 0.35031262040138245, -0.06089085340499878, -0.6957059502601624, 0.11315130442380905, -0.9229999780654907, 0.12479588389396667, 0.4085964560508728, 0.5445188879966736, -1.207930564880371, 0.3924185335636139, -0.05080721527338028, 0.14432001113891602, 0.44351476430892944, 0.23427774012088776, -0.024344032630324364, 0.5625667572021484, 0.09798575937747955, 0.39055368304252625, 0.5891896486282349, -2.308898448944092, -0.8978605270385742, -0.3150012195110321, -0.022128865122795105, 0.08508802205324173, 0.051658596843481064, 0.01401459239423275, 0.09794192761182785, -0.22027701139450073, 1.6072932481765747, 0.7641484141349792, -3.5850908756256104, 1.7137043476104736, 0.5658951997756958, -0.1659846007823944, -1.2512611150741577, -0.004505670163780451, -0.3229089379310608, -0.07856710255146027, -1.3429577350616455, -0.5023049116134644, -0.03578946366906166, -1.548660159111023, 0.12305455654859543, -1.8444416522979736, -1.0229032039642334, -0.03515317663550377, 0.8700761795043945, 0.07350253313779831, -0.559831976890564, 0.614051103591919, 0.9944966435432434, 0.30348002910614014, -0.2919491231441498, 0.12712205946445465, 0.4399922788143158, 0.05140235647559166, -0.1423017978668213, 0.06059961020946503, -0.7685956954956055, -0.9183580875396729, -0.1602238267660141], [0.0024262494407594204, 1.2165181636810303, 0.05279422923922539, 0.8617423176765442, 0.011725257150828838, -0.2768847644329071, 0.8174107074737549, -0.3240697681903839, -0.10243581980466843, -0.3223848044872284, 1.2124521732330322, -0.10569414496421814, -0.6529169678688049, 0.4847312867641449, -0.9565147757530212, -0.1976020634174347, -0.10329896956682205, 0.013709335587918758, -0.21995270252227783, 1.059130311012268, 0.003568568266928196, 0.038047704845666885, -0.4755357503890991, -0.17336541414260864, -0.02433171309530735, -0.769421398639679, 2.3957011699676514, 0.5072180032730103, -0.964633584022522, 1.3054726123809814, -0.34216856956481934, 0.11649665981531143, -0.49693483114242554, 1.3371585607528687, 0.602784276008606, 0.01473806332796812, 0.6433862447738647, 0.442818820476532, 0.12733349204063416, -0.46973690390586853, -0.20877113938331604, -1.2736736536026, 0.057326119393110275, 0.4528957009315491, 0.20718517899513245, -0.4511333107948303, -0.5294244885444641, 0.9068896770477295, 0.2841792404651642, 0.10022897273302078, -0.10310022532939911, -1.6384607553482056, -2.282180070877075, 0.012561596930027008, -1.08930242061615, 0.14454887807369232, -0.05231950059533119, 0.7742814421653748, 2.664339065551758, 1.0965100526809692, 0.2740996479988098, -0.12256311625242233, 0.29538393020629883, -0.11860127002000809, 0.025004878640174866, -1.0514415502548218, -0.0023637579288333654, -0.7083349227905273, -0.7509453296661377, 1.8910106420516968, -0.26139402389526367, -0.5520163178443909, 1.049770474433899, 0.20844067633152008, 0.006096185185015202, -0.5954452753067017, 0.05202227458357811, -0.47108855843544006, -1.5549228191375732, 0.014302821829915047, -1.5504565238952637, -4.239687442779541, -0.4503062069416046, -0.13876233994960785, -0.0032181874848902225, -0.1335856020450592, 0.12467768043279648, 0.07284919172525406, 0.12102717906236649, -3.026888847351074, -0.3635860085487366, -1.0079867839813232, 0.01214607059955597, -0.5714898705482483, 0.8656589984893799, -0.31680482625961304, 0.21349245309829712, 0.030715128406882286, 0.9982240796089172, 0.05697787553071976], [0.07536480575799942, -0.5620504021644592, 0.010773155838251114, -0.4497428238391876, 0.015362239442765713, 0.21933844685554504, 0.3810558021068573, 0.13247141242027283, 0.20893195271492004, -0.19996562600135803, 1.561299443244934, -0.2842944264411926, -0.4844803214073181, -1.057719349861145, -0.5617536902427673, -0.12733101844787598, 0.053489457815885544, 0.03164351359009743, 1.9118446111679077, -0.40241432189941406, 0.6214408874511719, -0.12461518496274948, -1.2019556760787964, -1.069549322128296, -0.2278575748205185, -0.6973763704299927, -0.09441117197275162, -0.4464234709739685, -0.6403818130493164, -1.2846558094024658, 1.3430746793746948, -1.213169813156128, 1.5301313400268555, 0.34643515944480896, -0.21442510187625885, 0.6725302934646606, 1.2889305353164673, 0.19293898344039917, 0.15532764792442322, 0.5816904902458191, 1.70689058303833, -0.36412668228149414, 0.03471847623586655, -0.20646366477012634, -0.09439954906702042, -0.2038755714893341, 1.0122612714767456, -1.1897064447402954, -0.8540260791778564, -0.039796024560928345, -0.0841975286602974, 0.3782251477241516, 1.1632304191589355, -0.0369131937623024, 0.9168955087661743, -0.04210953041911125, 0.42313942313194275, -0.3989315629005432, -0.5593292713165283, -0.31825923919677734, 1.4114009141921997, 0.07377500087022781, 0.09854096174240112, 0.05911003425717354, -0.052538610994815826, 0.8667705059051514, -0.02740701474249363, -2.16546630859375, -0.7362121343612671, 0.7348852753639221, -0.4846898019313812, 0.45818251371383667, 0.16741494834423065, 0.6023078560829163, -0.007278719451278448, 0.18396581709384918, 0.015205930918455124, -0.44343870878219604, 1.2013849020004272, 0.16130316257476807, 1.403496265411377, 0.46640437841415405, -0.017084181308746338, -0.6462723612785339, -0.04012728109955788, -0.32840877771377563, 0.2314758002758026, 0.852964460849762, -0.18462611734867096, -0.4528706967830658, -0.4576849341392517, -0.1312943696975708, 0.5992333889007568, 0.3083082139492035, 0.6303844451904297, 0.010336890816688538, -0.07777747511863708, -0.2885247766971588, -0.42112258076667786, 0.11820820719003677], [0.050511278212070465, 1.094030737876892, -0.005297399591654539, 0.8025617599487305, -0.07864664494991302, -0.12648245692253113, -0.742505669593811, 0.02168981544673443, 0.4464472234249115, 0.901543140411377, -1.5406023263931274, -0.04800577834248543, 0.41758543252944946, -2.1449434757232666, 0.09170868247747421, -0.6166446805000305, 0.05031949281692505, -0.01928931102156639, 0.594099223613739, -0.7814221382141113, 0.6102785468101501, -0.9831655621528625, -3.670159101486206, -1.4925546646118164, -0.05989423394203186, -0.3241457939147949, -1.5979280471801758, -0.24589061737060547, 0.5123414993286133, -1.3082127571105957, 1.0985623598098755, 1.024046778678894, -1.1276737451553345, -0.4381956160068512, 0.09237976372241974, -0.856766939163208, 1.4069632291793823, 0.05650653690099716, -0.31519144773483276, 0.3226615786552429, -1.1707446575164795, -0.8222362399101257, 0.022394515573978424, -0.5328708291053772, -0.17765885591506958, -0.6921542882919312, 0.3007633090019226, -0.6128166317939758, -0.036686819046735764, -0.06719493865966797, -0.05176650732755661, -0.23628072440624237, -0.37362155318260193, -0.019856352359056473, -0.9551974534988403, -0.004285987466573715, 0.13948357105255127, -0.41466447710990906, 0.5200966000556946, 0.16598142683506012, 0.658050537109375, -0.1187099814414978, -0.09580016136169434, 0.10273780673742294, 0.03662744164466858, -3.187978982925415, -0.09579308331012726, 1.0847247838974, -0.8332199454307556, -1.7571665048599243, -0.6724531054496765, -0.0816211998462677, -0.04078584909439087, 0.4834456145763397, -0.009193537756800652, -0.5293479561805725, -0.13979990780353546, -0.21117188036441803, 2.0653741359710693, -0.022822488099336624, -1.31707763671875, -0.44510552287101746, -0.6259706020355225, -0.6747369766235352, 0.02775135636329651, -0.20625442266464233, -0.07168654352426529, 1.1884065866470337, 0.40973415970802307, -0.21498358249664307, 0.19259795546531677, 0.04957425221800804, -0.19746895134449005, -0.2971668839454651, 0.07238706201314926, 0.5176241993904114, 0.06733576953411102, 1.2763848304748535, 1.2234779596328735, 0.693608820438385]], "dtype": "float32"}, {"__nd__": [[0.9955442547798157, -0.101956307888031, 1.2779748439788818, -1.1274852752685547, 1.8321948051452637, 0.15240424871444702, 0.13908004760742188, 0.28195154666900635, -0.03207427263259888, -0.8345129489898682, 2.4385411739349365, -0.11781531572341919, -1.695975661277771, -1.8116047382354736, 0.5275745987892151, -1.1147301197052002, -0.21343380212783813, 1.690577745437622, 0.5270119905471802, -0.5178322792053223, 0.8255347609519958, 3.149207592010498, 0.8051280379295349, -3.9540748596191406, 2.2606444358825684, -1.391481637954712, 0.38663357496261597, 0.4929395318031311, 0.4552488327026367, 0.7253182530403137, -1.700911521911621, 0.5824963450431824, 1.895074725151062, -1.8627909421920776, 1.2631289958953857, 0.6944677233695984, -2.801598072052002, -1.8823139667510986, 0.26621341705322266, -0.39887338876724243, 1.433504581451416, 2.5038156509399414, -2.9427695274353027, -1.559796929359436, -2.5240097045898438, -1.6848869323730469, -1.2148118019104004, 0.7311251163482666, -0.52345871925354, -0.39094609022140503, 1.2323530912399292, -1.289151906967163, 0.5038332343101501, -1.0470209121704102, -0.8037525415420532, 0.022638261318206787, 3.987679958343506, -0.06814047694206238, 0.08399257063865662, -1.3763723373413086, -1.1329289674758911, -1.0950331687927246, 1.6849606037139893, -0.31207275390625, -0.6816442012786865, -0.4451908469200134, -1.1832315921783447, -2.058129072189331, 1.1870825290679932, -1.0358591079711914, 1.0164011716842651, -0.8438852429389954, -1.4600651264190674, -1.1236406564712524, -0.8378405570983887, -0.3685592710971832, 1.1928067207336426, 1.072053074836731, -0.5642878413200378, -2.0211825370788574, 0.06077390909194946, 0.38229817152023315, -1.231865644454956, 0.07252475619316101, -2.089999198913574, 1.1904096603393555, -0.9969673156738281, -0.35199636220932007, -0.631696879863739, 0.5930889844894409, -1.0512775182724, 1.200385570526123, 0.7140888571739197, 1.2465659379959106, 1.0740165710449219, 1.1216013431549072, -2.0454084873199463, -3.2935266494750977, 2.2160754203796387, -2.217712163925171], [-0.7543246746063232, -1.6784119606018066, 1.1200041770935059, 0.9420672655105591, -1.9503369331359863, 2.312805652618408, 0.391477108001709, 0.8324476480484009, 1.4299699068069458, -2.9594979286193848, 0.851715087890625, 0.44916510581970215, -0.6151871681213379, 2.015286445617676, -1.811889410018921, 1.8308916091918945, -0.7107925415039062, -0.7435833811759949, 0.5661547780036926, 0.9699850082397461, 0.6191849112510681, -0.7419954538345337, -1.9692988395690918, 0.8513317704200745, 2.0304787158966064, -0.5154463052749634, -1.7640900611877441, -1.7229127883911133, 0.26414620876312256, 0.4454687237739563, -1.538175344467163, -0.9826483726501465, -0.5710610747337341, -0.8218918442726135, -0.45376160740852356, -0.17252743244171143, 0.22266606986522675, -2.2034287452697754, -1.2623554468154907, -1.3636274337768555, 0.11652320623397827, 1.6115013360977173, 1.0532140731811523, 2.3988819122314453, -1.0857652425765991, 0.7812700867652893, -1.074642300605774, -1.6145784854888916, -0.9367479085922241, -0.6898707151412964, -0.8052177429199219, -0.39811429381370544, -0.3179246783256531, -1.6381443738937378, 0.5981142520904541, 1.2291964292526245, 1.771672248840332, 1.5544353723526, 2.458134174346924, -1.3634023666381836, 0.5976440906524658, 0.22040855884552002, 0.26660096645355225, 0.8030222058296204, -0.11634296178817749, 0.07127037644386292, 2.081294298171997, -1.5843076705932617, -2.029033660888672, -2.7858855724334717, 1.3163902759552002, -2.3842923641204834, -0.390802800655365, 2.3385801315307617, 0.6652510166168213, 1.0588161945343018, -1.5639201402664185, -1.7330325841903687, -0.41584116220474243, 1.0670228004455566, 1.564485788345337, -0.07673667371273041, 3.2211074829101562, 3.0242886543273926, -1.1814347505569458, -1.8430626392364502, 0.7069295644760132, -0.7176154851913452, 1.1823331117630005, -1.0870463848114014, -1.1763243675231934, 0.4167877435684204, -0.5042508840560913, -1.364670753479004, -0.13242104649543762, 0.4113774299621582, -0.2846919298171997, -2.4110946655273438, 1.470577597618103, -0.3991709351539612], [-0.06854104995727539, -1.9657001495361328, -1.683477759361267, 1.6457152366638184, 0.12803351879119873, -0.858130931854248, -1.4122309684753418, -1.6871778964996338, -0.6891170740127563, -0.46384209394454956, -1.315237283706665, 0.5817890167236328, -3.099513530731201, 0.7492367625236511, -2.2754368782043457, 0.8709094524383545, 1.3923020362854004, 0.35477033257484436, -0.29069259762763977, 2.0571177005767822, 0.0143173448741436, 0.050265148282051086, -0.33264654874801636, 0.18417641520500183, -0.120272696018219, -1.332353115081787, 1.3778924942016602, 1.2943285703659058, 1.5494234561920166, -1.2656221389770508, -0.516108512878418, -0.1899462342262268, -0.4941120445728302, 1.2572978734970093, 2.5152788162231445, 0.024801820516586304, 1.3493211269378662, -1.8324213027954102, -0.9572974443435669, 1.872318983078003, 0.40908733010292053, 3.0209832191467285, 0.5463800430297852, -1.2052873373031616, -1.8457783460617065, -0.8832765817642212, 1.0698926448822021, 1.2412431240081787, -0.6946255564689636, 1.3935444355010986, 0.5890220999717712, 1.505985975265503, 3.171234369277954, 0.8655451536178589, -1.1858882904052734, 1.8567705154418945, -0.2430252730846405, 2.0917108058929443, -2.904527187347412, 1.7049552202224731, -0.5325055122375488, 1.2500123977661133, 0.9476879835128784, -1.8878153562545776, -0.21255415678024292, -0.7834489941596985, 0.022869281470775604, 0.7149378061294556, 2.041761875152588, 1.5046734809875488, -0.20560309290885925, 2.3815550804138184, 2.530867576599121, -0.39890730381011963, -0.9216574430465698, 1.9992520809173584, 1.0603479146957397, -0.6224896907806396, -1.318117380142212, -0.4366419315338135, 1.603837251663208, 2.7072248458862305, 0.8052549958229065, 0.4200930595397949, -0.11074948310852051, 0.2897528111934662, 1.2278568744659424, 0.09561896324157715, 0.2386181354522705, 3.3878676891326904, 1.443076491355896, 1.47349214553833, -0.049334049224853516, 1.8149996995925903, -2.2835264205932617, 0.9377931356430054, -1.0270061492919922, 0.09866289049386978, -1.6367756128311157, 0.14535276591777802], [-2.0311126708984375, 0.8664111495018005, -0.3277283310890198, -0.8193917870521545, 0.16003167629241943, 0.6485140323638916, -0.6280642747879028, 0.6579526662826538, 1.3408992290496826, -0.2744739055633545, -1.6157561540603638, 1.492908000946045, -2.194127321243286, -1.559691071510315, -1.2748794555664062, 0.5353845357894897, -0.6877898573875427, 0.7812061905860901, 2.41050124168396, -0.7456470131874084, 2.3785834312438965, -0.15705782175064087, -0.8021183013916016, 1.0839829444885254, -1.074503779411316, -1.152052879333496, -0.05180315300822258, -1.0867934226989746, 0.9812887907028198, 1.188155174255371, 1.932640790939331, 1.8870738744735718, 1.4514591693878174, 0.31076160073280334, -0.85359126329422, 1.079704999923706, 2.5788326263427734, -0.7616744041442871, -1.1140459775924683, -2.2119088172912598, -3.1908130645751953, 0.8239346146583557, 0.31568434834480286, 0.5241868495941162, 0.8023076057434082, -0.38080891966819763, -1.9515464305877686, -1.5534322261810303, 1.9914970397949219, -0.5278582572937012, 0.45890355110168457, -0.3316541314125061, -1.5420351028442383, -2.4264793395996094, 0.9522812366485596, -0.516028881072998, 1.875087022781372, -1.0281376838684082, 0.5817060470581055, -0.472097247838974, -2.6158697605133057, -0.7178171873092651, 0.30160975456237793, 0.8975971937179565, 0.4260627031326294, 0.616140604019165, 0.2529655396938324, 2.085120439529419, 1.9096325635910034, 0.5578506588935852, -0.3637039065361023, -1.8858072757720947, 0.38505202531814575, -1.0996593236923218, 1.0498238801956177, -0.589268147945404, 0.29568013548851013, -0.5590019226074219, 0.9715776443481445, -4.697823524475098, -1.385039210319519, -0.28412091732025146, 0.029145384207367897, 1.8665498495101929, -1.3174077272415161, 0.6795679330825806, 2.174777030944824, 1.0680714845657349, -0.347267210483551, 0.4835643470287323, 1.7330013513565063, 0.18310034275054932, -2.321972370147705, -0.9341211915016174, -1.5864076614379883, -0.02919142320752144, 0.3569393754005432, -0.8841678500175476, 0.6587486267089844, 0.287683367729187], [-1.3074986934661865, -1.6198163032531738, 0.15478229522705078, 1.4044080972671509, -0.786899209022522, -0.3591884672641754, 1.1754462718963623, 0.1034703254699707, 2.7520029544830322, 1.18855619430542, 1.5313259363174438, 0.2421283721923828, 1.8164293766021729, -3.037886142730713, 0.19990384578704834, 2.49031400680542, -0.6214573383331299, -0.4573884606361389, 0.7194510698318481, -1.390698790550232, 2.243541955947876, -1.190152883529663, -2.3524606227874756, 1.4529123306274414, -0.271279513835907, -0.5143195986747742, -0.8421268463134766, -0.5749491453170776, -0.7540582418441772, 1.162123441696167, 1.5183196067810059, -1.5299407243728638, -1.027417778968811, -0.37753820419311523, 0.3532146215438843, -1.3211236000061035, 2.703733444213867, -0.21425727009773254, 2.1712822914123535, -1.178446888923645, 2.1020612716674805, 1.786999225616455, 0.1955786943435669, 1.2994271516799927, 1.4502582550048828, -1.2417447566986084, -0.5569274425506592, -0.7685471773147583, 0.08216863870620728, -0.8560550212860107, 0.2709932327270508, 0.19899895787239075, 0.47571617364883423, -1.2536662817001343, -0.9528533220291138, -0.05044661462306976, 0.5936746597290039, -1.0264486074447632, -0.5195263624191284, 0.2364816516637802, -1.1714195013046265, 1.1093775033950806, -0.2816322147846222, 1.4984354972839355, -0.2852916419506073, -2.176600217819214, 0.8492220640182495, -1.003198266029358, 2.075840711593628, -1.28114652633667, -0.4846559762954712, 0.3226630687713623, -0.09009945392608643, -0.8477635383605957, 1.2735958099365234, 1.6285629272460938, -2.6109800338745117, -0.2556838393211365, 1.6042852401733398, 0.6384167671203613, 1.2483882904052734, 0.2604300379753113, 1.025692105293274, 1.8717421293258667, 0.8750889301300049, 0.40992921590805054, -0.646895706653595, 1.4292951822280884, 0.7402191162109375, 0.22048185765743256, -0.7004408240318298, -0.0664028525352478, 0.7349323630332947, 0.8647807836532593, -0.17496764659881592, -1.404000997543335, -0.29680687189102173, 3.7568230628967285, -1.838208794593811, 1.6213219165802002]], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.5], [0.5]], "dtype": "float32"}, {"__nd__": [[0.25], [0.25]], "dtype": "float32"}, {"__nd__": [3.0], "dtype": "float32"}, 1e-06], "e": {"__tup__": [{"__nd__": [[2.99999737739563], [2.99999737739563]], "dtype": "float32"}, {"__nd__": [[0.75], [0.75]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(add_rms_norm, json.loads(_TESTS), 'triton', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")